In [1]:
from collections import Counter
from io import StringIO
from Bio import SeqIO
from Bio.SeqUtils.ProtParam import ProteinAnalysis

# FASTA data
fasta_data = """
>lcl|NZ_CP026520.1_prot_PC41400_RS00005_1 [locus_tag=PC41400_RS00005] [protein=ATP-binding protein] [pseudo=true] [partial=5'] [location=complement(join(6508155..6509090,1..>3))] [gbkey=CDS]
VLGFHKESILFKRIEDIFASVPPEHRLVERSLLDGRIVRNHAVSWMNNQERYELLLDSNLLRDNEDQIVGAYIIFKDVTN
LRSLEAQVQRSDRLAMIGQIAAGTAHEIRNPLTSIKGFLQVLKKTFESKDMAKEQGYIDVMLDEINRVNELVNEFLLLGK
PKNVLLQKIDVSTVIKGILPIINNEAILHKVHMNYEESSGLPPVIADQELLKQVFLNICKNGIEALTDGGTLTIVERLDP
DGRFVNIEVHDTGPGIPVYVIDKIFDPFFTTKEEGTGLGLSVCQRIIHDMGGHIRVSSKGYGTTFTVSIPCT
>lcl|NZ_CP026520.1_prot_WP_241688758.1_2 [locus_tag=PC41400_RS30135] [protein=SWIM zinc finger domain-containing protein] [protein_id=WP_241688758.1] [location=66..965] [gbkey=CDS]
MHASVRGNDRYHTTIDLESFSAGDCTCSFDKPCRHMAAVIFALYVPHGRPELLVQELKRAMTTKRKSAKNADRESKRRTS
EPVPGGSPEDWHRFFEAQYHGFSISHQYSFDLFVTSAFERFQAFASGWPDGIRLLYDLHVNFFVMHRMEQFFGDTKNSYL
SGYHESNAKTAAADCLSRLDRLLASPALPVSVKEHPKRWTEALNRLASLALQGKEGPNDWLYAYRAVWWLCRRKRNGSSA
RKNGWISSSRNGRKAAGTASRTFCLWPGLISTSWRTTRTRRFLDWRLCISAEQEILSSI
>lcl|NZ_CP026520.1_prot_WP_241688759.1_3 [locus_tag=PC41400_RS30140] [protein=hypothetical protein] [protein_id=WP_241688759.1] [location=962..1507] [gbkey=CDS]
MRRCHERAEWDILLSWLRWLAPTMQSAAQDDFRLLCQYWTDAAKHADADDEWVRIMETLLPRSYYVYTGYLLQTKRYRQW
VDLQIANRIGPQNLYALELRSVEESDPSLLLPLYHQAVERSVLEKNRTSYKTAVRLLKKLHGYYTTLGRKREWETYINRL
SDHFVRLRAFQEELEKGSWLP
>lcl|NZ_CP026520.1_prot_PC41400_RS00015_4 [locus_tag=PC41400_RS00015] [protein=DEAD/DEAH box helicase] [pseudo=true] [location=1504..4382] [gbkey=CDS]
MIETSPLTVHAVWMPSGAFFVWGTRENGIWDAQDLKGLLFAWHAPSFYGTFIETTEWGHREGLSLPALTALDFFADPHTV
KHLSIRWDTDTANLRRLAPEVKEALAGGRFVPITSSGRRAP*AGSCSSKERPPRPLPRRRSCGSTSSSPPGSKPTAPAKR
RSAGSKRRIRCCGWVHVLPILGWTKRTGSCRSAGSTIRRRSAPASSSWNPRGTRTGGCKSCCRTDPTTRSCAP*LRRASR
WKASWPCRSPGLRTSDASRGMSADGCAFCRG*TRRTKESRRSSARA*ATRRRGASSRKAACGSWRRASPSCCPPGGNGRA
SCAPSSRPSSKIRLAAEAASRCSGLPSLCSLTGASPSTISS*ARKNSSNCSKRSASSSASAAAGSSSTRSSCAKRRSCWP
ERTGSRDLRCARCWSSSCWAAVRAESRRDRRTKPPIPTTGRLRSRSS*PASSAGCSSSSRARQSCPASIPRLLSWGRCVL
ISWRAARGCSSCAASAWGLVLLTTWVSARPSSTSPIFFR*RKPKLRKRLPCSSAQPP*SGTGRWSCAVLRRTSAFTFITG
RSAPKEASSRNP*KARTSSSHRIRFPTSMRRNCARCPGAASVWTKPRMSRIRTRSKRPRSAVSRASTGSP*RVHRWKTG*
RSCGPSSISSIPAISARSRSLRGGSSRRSRNPATAV*SPAYSASCGLS*CAASRKTPPSSSIFRRRTSPRHS*RLRPSRV
RSTSITSVTSSRKSTRCLRWNAEGIFLPL*RSSSRSATIPRSLCARDPAPIGATAPVKWSVCWTWCRSCASRATSA*SSL
NSWRLAIFSRKFCKASSASPSPSCTEALRGRQGMR*SPVFRTQSFPWKSSSPSFCCRLKQAEPV*T*RPPITSFILTAGG
IPPWRTRRRTVLSGSDRRGTCRCTNSLRSVRWKRKSMK*SRESSV*ASKLWAAGRIGLRSCLRTSCVICLLCGWNG*MG
>lcl|NZ_CP026520.1_prot_WP_042235136.1_5 [locus_tag=PC41400_RS00020] [protein=ATP-binding protein] [protein_id=WP_042235136.1] [location=5004..7883] [gbkey=CDS]
MKFNRSRLSLSYVIAIVLLLALVGTSSYISSLNMERQSDAVIENAIPVISAANNLLQDLLNQETGIRGYLISSDPKYLEP
YENGKAALQKDLEVIRTLESGHDDLRILIQTQVNPHIQKMLTFYTSQIELVQSGKVQEAQGRINIGKQYMDQYRTLHLSI
LDVLNKISDSSHDSIKRAGAVSRMIILSGGILGILIGIWSGLIQLRASRAESQLRKSEETYRFMAESLEAQNEEIVAQQE
EQQITLAKLSDRERELELISNYQEKLTGFTELKQFLQHSIPALLDSVSYDAALVVLKCKEAEHYEVLFSSGYPRSFASRA
QSELYGASQRVFDEGCAFVQKREATEAERGFHDGISVAVDHYFPLQDDRQQVIGFLLLTNFHGFDRHEEKERLTLGLIRQ
FDLAFYAQIMNDERRKQAAHLEELNDQLVLEKQYIESQRDLIEGILEAAHEGMMMCDRNGKILIANNQMHKFFHLDEQIG
HDFVDSCQEIAASSSGFTGVSQSIKALFDGDINQLNERFTFMNGEEPQFTEIYATSVGENDSDESGYLFVFRDRTEEERI
DAMKNEFVSIVSHELRTPLASVLGFIEILIHRELPREKQQKYLDTIFKEATRLANLINDFLDLQRMESGRQVYHFAPVDM
KELVYDIVEKWQDKQNHDIGISVLSPAEVWVRGDSDRLRQVVQNLLSNAIKYSPGAERIDLQIRESGGSVFIDVTDYGLG
IPEEAKEHLFSKFYRVDNSDRRQIGGTGLGLAIVKEIVEGHQGKLTFRSQMGKGSTFTVELVPFEMVSLNNKIVIFEDDD
NLAKLIQVALAKLQLPSVQLRSAEEGILSLKRKTTEAPLLFIVDIHLEGSLTGWDFISELYHHPVYNRTPVIVSTALEPP
LDYHEKDIEKFMRKPFTMERLIQVAQDLLSNRDKPSYVFPSQDRNLITSSLERNGIKVHDVRVNADTIEFEIETEDEEK
>lcl|NZ_CP026520.1_prot_WP_042235134.1_6 [locus_tag=PC41400_RS00025] [protein=methyl-accepting chemotaxis protein] [protein_id=WP_042235134.1] [location=complement(8239..10404)] [gbkey=CDS]
MKKEMLTKSFKGLNRNSLKVKIPLLLLILALVPLLVSEYFLSNQFTSVLRKSIEENQVKITTYTADYISSTLVQKVSTVQ
SVIQNHTEMGTQTEAQKIAILKTMSENDTEVKIALFADKDGQAVDQNGKKSSLANDETFKSVKDTKKRVVSDIIKDSSSN
QYSLKVSAPVIGANGEFAGMLQTSINPSRMINQVNKVKIGETGYAYLMSKEGIYMVHPQTDRIGKPLSDMATPERLELFK
KTVLVKEQGQVAYKEKDNTNKAAVYEEIKVPGWRIVVTGNSDEIYSQVDQVNFVAKIIVLVSAIVVVAAAIWLSTFILKP
LLGISAMMKRIAGGDLSRRLKVQSQDEMGDVQQNINEMLNSFSALLQKLSMAIEHMAASSQQLSAIATQSADVSQNVNQT
VLVMVKGAETQFETAEQTTEAMNEMAAGVTKIADSASTVSEVAQGAVQRVNQGSRDMQDAIHQIQQASEAVEATSLTIKA
LEESSARIHQVVGFISEIAKQTNLLALNASIEAARAGEHGRGFAVVAQEVKNLAEQTSGATVQINDTIKQVLSATKQAAA
SMSHGKAEVDRGVEQVRNTDRVFEDILETVSSINGQIQEVSAATQQISAGTEEVAASMQETVQISRDSLGKMNDMSRSTE
EQHAGMKEIVISASELNRLASELQELSGKFTVERNAAEDNETEDKEAGNKKIKQPKQKRQKPDKPIKGAKKGRAGKGNTE
K
>lcl|NZ_CP026520.1_prot_WP_164977106.1_7 [locus_tag=PC41400_RS29605] [protein=hypothetical protein] [protein_id=WP_164977106.1] [location=17615..17785] [gbkey=CDS]
MENRTHSGNYKGTTHMNAENQDMEFANDTLELAEDKRAIRNKDKLNTQAPDAKTQD
>lcl|NZ_CP026520.1_prot_WP_042231213.1_8 [locus_tag=PC41400_RS00115] [protein=hypothetical protein] [protein_id=WP_042231213.1] [location=complement(17982..18653)] [gbkey=CDS]
MEYIHITDINDPLFAKMHQLMQTVFPAEEVLDFDLWKEPLEDPGIRVFVAVHEGEVVGATEYRYYTDLGIAMTDFTIIGR
PGLGIGRFLSRKRQHDLADLAAQSGKPMIGMFAEIYDPYRIEDHSFGGVKPMDPFVRREVLSHLGYRRLDFPYVHPSWQN
DGEAVSELDLCFMPSDENAEYVPSSLIADFLTRYYSVLENKPPTWQEMISRLRESERTALLPL
>lcl|NZ_CP026520.1_prot_WP_042231214.1_9 [locus_tag=PC41400_RS00120] [protein=GNAT family N-acetyltransferase] [protein_id=WP_042231214.1] [location=complement(18723..19415)] [gbkey=CDS]
MNATPKSEPPKYMKKMYVFDGEKPVPALIRNYTEADFEALIDVQRLSFPPPFPEELWWNEEQLRSHVRLFPEGALCVEVG
GKIVGSMTALRIRFDPLHPEHTWEEITDGGYIRNHDPEGDSLYVVDVCVMPAYRKLGLGKWLMFSMYETVVHLGLRRLLG
GGRMPGYGAQAGKMTADQYVDAVVAGVLHDPVLNFLLRCGRMPVAVLPDYLKDEESLNHALLMEWRNPFL
>lcl|NZ_CP026520.1_prot_WP_042231216.1_10 [locus_tag=PC41400_RS00125] [protein=carbon-nitrogen hydrolase family protein] [protein_id=WP_042231216.1] [location=complement(19493..20350)] [gbkey=CDS]
MKFRVSAVQHHLHTISSFQEFADQVEHYVKTAQEFDSDFVLFPELFTTQLLSIRNPGEGALTINDLPSFTDSYMELFKGL
AKSTGMHLIGGTHIIEDGGRLYNSAFLFYPDGRVGEQRKLHITPTEVKEWNMGAGDVLQVFDTDKGRIAILVCYDMEFPE
IVRMAKARGADVIFCPSCTDDRHGFHRVRYSCHARTIENQIYVVVTGTIGSLPTVDFMRGNFGQAAVLTPNDVPFPPRGI
LAEGEINDDMIITADLDLALLEEVREKGSVTTWRDRRTDLYTDWK
>lcl|NZ_CP026520.1_prot_WP_042231217.1_11 [locus_tag=PC41400_RS00130] [protein=class I SAM-dependent methyltransferase] [protein_id=WP_042231217.1] [location=20466..21260] [gbkey=CDS]
MQKQFSEHNQTAWNQHAYEAWLNRFGTPEEAAALISRNPEGTVRSFSRFLGDLSGIKAVNLLGSHGSKAAALALLGASEV
TVVDIAAENAKYGTELAEAAGAHVRYVVSDVLELPEEELTGDYGLALMEFGILHYFLELKPLFDVVVKLLASGGRLILQD
FHPVTTKLISSRGSTAKVRKHKVTGDYFDTSIEETDVAYSKFLPGQDEAALIKVRHRKWTLGEIVTAAASSGLFIEVLEE
EPNRSSDVYDKGIPKSFTLVARKL
>lcl|NZ_CP026520.1_prot_WP_042231219.1_12 [locus_tag=PC41400_RS00135] [protein=MerR family transcriptional regulator] [protein_id=WP_042231219.1] [location=21589..22050] [gbkey=CDS]
MTRLKIDDVAKETGLTKRTIRYYEEIGLLEAPTRSEGGVRLYTQENVDRLKKVLLAREVLGFSLQELQHFLAIDTLLASY
REGSDGYAVELKKQEELKRVAGGLAEQVDMLDLKIHKMEAFRKDMEELLSKVREAIGQPQTAASSGKIPSSGQ
>lcl|NZ_CP026520.1_prot_WP_042231220.1_13 [locus_tag=PC41400_RS00140] [protein=MFS transporter] [protein_id=WP_042231220.1] [location=22106..23284] [gbkey=CDS]
MDQGIKKQTRARYVTVFATFLAFMGIGVVDPILPIIAEKIGANPAQIELLFTAYILTMAIMMIPSGLAATRFGDKKMMVA
GLAVVTVFAFLCGASGGITELALFRAGWGFGNSMFFATAMTLLIALTPSANTAVGMYEAAIGLGMAGGPLVGGLLGGISW
RLPFIATGCFVLIAFLLVLFMIQEPEKKSVRKAAGLKELLHLATHKPFLRVAGSSMFYYYGFFVVLAYSPLMLHMSAIQL
GLVFFGWGLMLAYGSAKLAHNLEERYKPSQIIPWSLGAFCLILLLLFLVHTLWLQILLIIVIGLFCGLNNALFTSHVMEV
SPFERSVTSGAYNFVRWLGAAIAPVLSGYIAHHLSAQLPFAVAAGLALIAVIAMSVWRSRPAKAGAAGPQAG
>lcl|NZ_CP026520.1_prot_WP_042231221.1_14 [locus_tag=PC41400_RS00145] [protein=UDP-glucose--hexose-1-phosphate uridylyltransferase] [protein_id=WP_042231221.1] [location=complement(23371..24984)] [gbkey=CDS]
MSMDLEREKARNQADQAALAVERLVSFALAKGMIESLDVCAARNALLDALGLPEPYQGADGNAVDETVAAEPGEMLEVLL
DFAYFRGRLEEDSTTYRDLFDARLMGLLMPRPSEVARQFGEITREHGVRKAADWFYELCINSNYIRMDRIRKNGYWLAPT
EYGNLEITVNLSKPEKDPKEIALLRTLKQSGYPQCLLCVDNVGYAGRLDHPARQNLRVLPLTIDGEAWFFQYSPYVYYNE
HCIVLNEKHVPMKISERTFRRLFDFIEQIPHYFVGSNADLPIVGGSILSHDHFQGGNHKFPMELAPVERKFASDTHPGVT
VGIVRWPMSVLRLEGSGREDVLRAADMLLAAWRAYSDPSADIAAFTQTADGVRVPHNTVTPIARFNASGQYELDLVLRNN
RASEEHPDGIFHPHRNLHHIKKENIGLIEVMGLAVLPGRLQEEISGIAGFLTGDSAWSGADGLEAGHPLEKHAAWIGELV
AAGGTALSLQEAHELLQREIGGKFKQVLEDAGVFKRTDAGQTAFGRFLGSLGFVHAE
>lcl|NZ_CP026520.1_prot_WP_042231222.1_15 [gene=galE] [locus_tag=PC41400_RS00150] [protein=UDP-glucose 4-epimerase GalE] [protein_id=WP_042231222.1] [location=complement(25031..26023)] [gbkey=CDS]
MAIMVTGGAGYIGSHAVAELLDRGEEVVIVDNLQQGHRDAVLGGKFYEGDLRDEAFLDTVFRENEIDAVIHFAANSLVGE
SMTDPAKYYHNNVYGTLCLLEKMKKFDVKRIVFSSTAATYGEPENTPILETDRTEPTNAYGETKLSMEKMIKWFDRAHGI
KSVSLRYFNAAGAHESGRIGEDHHPETHLVPLILQVALGQRSHISVFGEDYATPDGTCIRDYIHVSDLADAHVLAVEKLR
QGGESGVYNLGSGNGFSVKEVIDIAREVTGHPIPAVLEARRSGDPAVLIASSDRARQDLGWKPKRDNLQAIIASAWAWHQ
AHPEGYADKK
>lcl|NZ_CP026520.1_prot_WP_042231223.1_16 [locus_tag=PC41400_RS00155] [protein=galactokinase] [protein_id=WP_042231223.1] [location=complement(26108..27283)] [gbkey=CDS]
MADLNLLKTQFIELYGGAEKDITVFHAPGRVNLIGEHTDYNGGYVFPAALTFGTTLLLRKNNEGLLRMASTNFAHKGSCS
LDDIVYREEDDWMNYPKGVVAKLQQKGQGPRSGYDMLYHGEIPNGAGLSSSASIEVVTGYGLLEAEGYPADRVELALLAQ
KAENEFMGVNCGIMDQFAVANGRKDHAILLMCDTLEYELVPFQSGSYKLVIGNTNKRRGLVDSAYNERRSQCEQAVRDLQ
TRFPDLTLLGQISLEQFEANKDIIKDDIVRRRAQHVVQEIDRVLKSIDVLKANDLEAFGRLMTGSHESLRDLYEVTCREL
DVMVEAALSVPGVLGSRMTGAGFGGCTVSLVHEDSIEAFREKVGSIYKEQTGLTGDFYVCSIGDGVNRVQD
>lcl|NZ_CP026520.1_prot_WP_042231224.1_17 [locus_tag=PC41400_RS00160] [protein=AraC family transcriptional regulator] [protein_id=WP_042231224.1] [location=27582..28448] [gbkey=CDS]
MKREPTYYVVSNPEPPEDAALHVLFSGESQTKPLHRLGPKVYDYFLIHTVISGRGRFTAAGRELLLGPGDSFCIAPEQLV
GYEADEADPWRYCWLAVTGAEAAAHMRKAGLTPEQPVIHTAHAGRRVPALFRRLQRALRHKSGTASLRALGYLHLLLAEY
ADAARPPLPEPGKAHSRSRAEVTVQQMLHYLSTQYAEDISIESMAEALGYNRAYLSRLFKEHTGQTSVSFLSKLRVDKAR
QLLRERLELTIEQIAFSVGFRDPLYFSKQFRRWYGLSPTDYRSDMKRM
>lcl|NZ_CP026520.1_prot_WP_042231225.1_18 [locus_tag=PC41400_RS00165] [protein=prolipoprotein diacylglyceryl transferase] [protein_id=WP_042231225.1] [location=28573..29349] [gbkey=CDS]
MTFPVYIDLGFIRLHPHLVFEALAYFIGFRLYLYTRSKEKLQASQALWVLVGATLGAAAGSKALYWLEDPALTLQRWNDM
TYLMEGKTIVGGLLGGLIGVEWTKKRIGHTRSTGDDMALPLLTGMMIGRIGCFLTGLDDHTYGIPTGWVTGVDFGDGILR
HPTQLYEIAFLLLLGITILVMKNRARHGSVHLPDGMSFIIFMGGYLLFRLLVDFIKPTPHPYAGLNNIQLACIAGLIYYT
LLLRRKLSAKPTLIGKGT
>lcl|NZ_CP026520.1_prot_WP_042231226.1_19 [locus_tag=PC41400_RS00170] [protein=radical SAM protein] [protein_id=WP_042231226.1] [location=29355..30767] [gbkey=CDS]
MPTKNRPYLFYELTNSICSQCLRKVEAKVIIEDENVYLLKYCGRHGREKVLISDDAEYYKRCREFLKPAEMPQHWNTPIK
YGCPYDCGLCPDHEQHSCLTLLEITDKCNLQCPICYAESSPKRETWRSLEQIETMLNAIVRNEEEPDIVQISGGEPTIHP
QFFEILDLAKTKPIKHLMVNTNGLRIAQDREFVRKLSRYMPGFELYLQFDSFEEEALRELRGADLRDVRRRAIEHLNEFN
ISTTLVVTLKKGLNDSEVGDIIRYGLQQRAVRGVTFQPVQVAGRTEAFDPAKDRLTLSEVRRMIIEQSGIFQPEDIIPVP
CHPDCLAMGYALKLGGQALPLTGMIDPQILLEGGRNTIVFEQDQTLKDQVFDLFSTAHSPESSALSLKNLLCCLPSVAVP
EHIGYDNVFRVLIMQFLDPYNFDVRSVKKSCVHIAHPDGRIIPFDTYNLFYRDGREALLEELRAETGGLA
>lcl|NZ_CP026520.1_prot_WP_042231227.1_20 [locus_tag=PC41400_RS00175] [protein=hypothetical protein] [protein_id=WP_042231227.1] [location=30764..31198] [gbkey=CDS]
MSDYKNDPTESGTQPEAGTSSESGKEKGPEPAWNSEPGDHATETGSEQTSSPPSHEHGGSGRTRRPWKGLDVSKGIGWVA
LGHLLWLFALPMYFGIGIVQMLYVLPLLLIFSKRTAVVQGILIGAGITFLLNAACFGYVMFSFG
>lcl|NZ_CP026520.1_prot_WP_042231229.1_21 [locus_tag=PC41400_RS00180] [protein=hypothetical protein] [protein_id=WP_042231229.1] [location=31250..32155] [gbkey=CDS]
MKNPLIAVFLLALASWIGNLWVYEKNQLPEPVFLSHFIDLPLEEGASFKLSYLENHRGLHIVDVEVPELPYASVNFLNKN
RYAYQNYQSVLISLEQQGQETDPSAAPLHIKQVRAWLSDGSSRMLDIGDVTLTPQAKNTADRSLFEFSSGGTSNQNVGYT
LVTTKEDLILTGVESPYPDDNYPRMKLYLDALSLEQEPGYRTSLQSGDKAEVEGLPVEEISFPVAVKKGSYMRLSYRLED
TGTDTPEHKAAEKYNRIEGAVRVKAQASDGFKDDMAGFIRYESDLSASDIRKIVQTRRELP
>lcl|NZ_CP026520.1_prot_WP_241688760.1_22 [locus_tag=PC41400_RS00185] [protein=hypothetical protein] [protein_id=WP_241688760.1] [location=32258..32761] [gbkey=CDS]
MAASALGELGKQEQAFRVARRFATVLIFLSLPNEINLGGALVTNQEFSFLWFSLGLVNSALLLAFIYYLCTGVISAAARS
SLPELAQSAHIRRSVFLAVTGATLILSPLALNMPEAWQFLLIALGIVGFLSGISIILLLRRAGQLLGPLKPAPDKRESII
SGEDENM
>lcl|NZ_CP026520.1_prot_WP_009675785.1_23 [locus_tag=PC41400_RS00190] [protein=ABC transporter permease] [protein_id=WP_009675785.1] [location=complement(32913..33872)] [gbkey=CDS]
MSLDILGQLINTTLVFSTALIFAALGGIFSERSGVTNIGLEGLMVSGAFAAAVSTFYAQEAGMGSMSPWIGLLAALVFGV
LFSLIHAVATITFKSDQVISGVVINFLAAGVTVYLVKLLFEGAGQTETLTEVFSKWAIPGLSQIPVLGYGIFTAYPTTYI
ALILVAVSYYVLFKTPFGLRLRSVGEHPSAADTVGVNVRRMRYIGVMLSGAFAGLGGATIALTTTSNFSHNTISGQGYIA
LAAMIFGKWHPVGALGAAVFFGLSQAIRNYVQLFEWSQSIPQEFIFMLPYVLTILVLAGAVGRAYPPAALGEPYDPGKR
>lcl|NZ_CP026520.1_prot_WP_042231231.1_24 [locus_tag=PC41400_RS00195] [protein=ABC transporter permease] [protein_id=WP_042231231.1] [location=complement(33873..34952)] [gbkey=CDS]
MNKVMKFFMRDSAIIPLAAIVLGFVFGAIVMLAGGYNPITAYSSLFSRVFGNMYDFGETIREITPLILTGLAVAFAFRAG
LFNIGGEGQFIIGMTGASFVGIKLGGLPAIIHAPLAIIVGALLGGLWAAIAGYLKAKRGVNEVITTIMLNWIALYLANYT
VNHFLLLKGQQRSVDIEPTASVSLGWLSAMFDNARLHWGTLFAILAAVFFYFFLWKTKQGYELRAVGHNTNAAEYAGMNV
NKNIIKAMFISGMFAGLAGVFEVLGVFHYQTVMPASPGYGFDGIAVALLGLTNPFGVLLAASLFGILTYGSAGMSFGADV
PPELIRIVIGSIIFFVAAQGIVRWVIAPLYFKRKKEKVV
>lcl|NZ_CP026520.1_prot_WP_042231232.1_25 [locus_tag=PC41400_RS00200] [protein=ABC transporter ATP-binding protein] [protein_id=WP_042231232.1] [location=complement(34945..36483)] [gbkey=CDS]
MSAATPVVELKQITKRFPGIVANDSISLKLQKGEIHALLGENGAGKSTLMNIVFGLYQPDEGGIEVNGQPVLIDSPNKAI
ELGIGMVHQHFKLVQPFTVTENIILGMEPKKGMNIDYKSAAEQVKKLSEQYGLQVNPHAKIQDISVGMQQRVEIVKTLYR
GADILIFDEPTAVLTPQEITELMDIMKRLVAEGKSIILITHKLKEIMQISDTVTIIRRGKVIDTVKTSETTTRELAEKMV
GRSVSFKVDKKPAQPKGPVLQVRDIKAKSHDGLPVLNGLSFDVKAGEILGIAGVDGNGQSELIEALTGLRKVDSGSIKLL
DKEIVNRSPRHISEAGVSHIPEDRHKHGLVLDFSMSENMVLQTYHRPAFNRGGFLNYEAIDKQAERLIEAFDVRTPSIHT
KARSLSGGNQQKAIIAREIDRNPDLLIAAQPTRGLDVGAIEFVHKQLIAQRDQGKAVLLISFELDEILNVSDRIAVIYEG
EIVGEVLPEETNDQELGLMMAGSMKREGVTHE
>lcl|NZ_CP026520.1_prot_WP_042231233.1_26 [locus_tag=PC41400_RS00205] [protein=BMP family ABC transporter substrate-binding protein] [protein_id=WP_042231233.1] [location=complement(36635..37675)] [gbkey=CDS]
MKKFFKLSLVMMLAFTLVLAGCGKKADNGGAAPSGNASGGEKKSDVKIGMVTDVGGVNDKSFNQSAWEALEKLKKDTGIS
TKYLQSKADSDYVPNLNQFVQGKYNLTWGIGFVIADAMKTVAQQNPTANFAIIDNVVEAPNVQSITFAENEGSFLVGVVA
AKMTKSNKIGFVGGMDIPVIKRFEAGFKAGVAAVNPDAKVLINYTGAFDKPDLGKAAAATMYNDGADIIFHSSGATGNGV
FNEANDRIKQGKKVWVIGVDKDQSLEFGDQVTLTSMVKGVEEAVTKYSKMVVDGNFKPGAPVTLGLKDNGVGLASTSSKN
VPADVLKLADEYKEKIIKGEIKVPEQ
>lcl|NZ_CP026520.1_prot_WP_042231234.1_27 [gene=ilvA] [locus_tag=PC41400_RS00210] [protein=threonine ammonia-lyase IlvA] [protein_id=WP_042231234.1] [location=complement(37922..39178)] [gbkey=CDS]
MSNPSTVTVEEIIRANHALSRVIEPSPLQRSELLSRQYDCNIYLKREDLQVVRSFKIRGAYNVIQSLTEEERSRGVVCAS
AGNHAQGVAYSCNHLGIEGKIFMPTTTPRQKISQVKMFGGPYVEVVLIGDTFDDSSTQARIYCERENKVFIHPFDDVRTI
AGQGTVGLEVLNQMPVSPDFIFTSIGGGGLAAGVGAYVKAVSPETAVIGVEPEGAAGMQRSMEAGRVVTLETVDKFVDGA
AVKQVGEMTFEICRRVLDDIVLVPEGKECTTILDLYNNNAIVIEPAGALPVAALDAYRSQIVGKNVVCVISGGNNDIDRM
QEIKERSLIYEGLKHYFLVNFPQRAGALREFLEEVLGPTDDITRFEYMKKTNRDNAPALVGIELKNKDDYEGLISRLCKK
DINYVEINNNPTLFNLLI
>lcl|NZ_CP026520.1_prot_WP_042231235.1_28 [locus_tag=PC41400_RS00215] [protein=efflux RND transporter permease subunit] [protein_id=WP_042231235.1] [location=complement(39370..42708)] [gbkey=CDS]
MRGIIRYSLNNKLVIWLLTILVTAAGIYAGLSMKQETLPNLEVPVLTVTTVYPGAAPEEVADRITGPLEQKVRSLDGVDV
VSSTSMENVSSIVMEYSYKKDMDKAVSEVKEAVADFVLPSGAQETKVSKININAFPVVSLSVAGDKQSLDELTRQIESDL
LPALEGVPGAASVEISGQHQQEVRLTFERDKLKTYGLSEDTVKGIVQGAAVQVPLGLYEVGRTEKTLVVDGNVKSLDDLR
NLEIPLVPGGAGAAGAAGPAAGGAGGAPGAHGGAASGGAPGAGAAAAPGSAAGAGRTGGAASAPAGAAGLPTVKLAELAA
IELADRADSISRTNGKESIGIQVVKAPDANTVDVVNAVKEEADKFRQANSGVETVIMLDQGKPIEDSVNTMLSKAVFGAL
FAVLVIMLFLRNWRTTLISIVSIPLSLLIGILLLQQMDITLNIMTLGAMTVAIGRVVDDSIVVIENIYRRMSSPSEPLRG
KALIREATREMFMPIFSSTLVTIAVFLPLGFVSGPVGQLFLPFALTMVFALLASLLVAITIVPMLGHSMFRRRLERMGPA
PVPAFAEGGASAALPLGMDPAESVRLRRVRDKGQNRRGHDETGRLGRFYRRVLDKTLNHKLIAFGTAILLLAGSLLLATR
VGVSFLPEEEQKYAMVTYTPSPGETLDDVKSKALEAEKYVLGRSGVRNLQYSVGGQNPMSPGPSKSALFYVEYDPEFADF
TAEKKRLVQGLTDTVHGSWKPMDMGGGVGGSKLSLYVYGEKLDEIRTAADKIQALMKNRPEFDKVASSLSESYGQYTIVA
DPAKLSKLGLTAGQIATAFSRDRERPVLAKVGVEGKEVEVYVQADKKTYADLSAIENSTITSPLGVEVPLKDVAKVEEGL
SPNAVTHRDTKLYAEVSADITSKDVGAASTGLKSEIDKLGLPSSVRVDFGGVTEQINETFSQLGLAMIAAIAIVYLVLVM
FFGGALTPFAILFSLPFTIIGAIVALLVTGETLSVSAMMGALMLIGIVVTNAIVLLDRVIHNEKAGMRTREALLEAAATR
LRPILMTALATVGALLPLAFGFESGGLISKGLAVTVIGGLISSTLLTLIIVPVVYEFLVKFRRRRPASELED
>lcl|NZ_CP026520.1_prot_WP_042231236.1_29 [locus_tag=PC41400_RS00220] [protein=S-layer homology domain-containing protein] [protein_id=WP_042231236.1] [location=complement(42916..45237)] [gbkey=CDS]
MKIILKAGTAVVTSALLFGTVTPVMASKAVTGASATIANPVSDQGKEQIDISQEEVKVTKEQAVELAKKYVALPEGFSLQ
GANLSMSRGFGGKPAKTWNLNFSHKEKKDYYSYLSYSLDADSGALLGYNFSSKNPDSKPVYPPKTSLDQAKTIASDYIAK
LSPELQKQVKYNDLRESSRKTPLNVYEDYRFSFDRVVDGVPFPQNNITVAVDSEGRLSDYRVYWDKDAAFDKKEGVITPE
QADKAFRNQAEPVLQYLIPYQSKELKPAISYSMAALQLDAKTGKPLYESKAPAAVKTLTEQPQSGTGTAVTTKESALKRA
QQLVKIPADAKLEEANYSEYENESTKQTDANWNFQWALPGKDGAKDNRGYVSLAIDAKTGELTNFSHYDYSSNRTIKPEE
VKVSFEEAKGKAVDFVKKTNPSYAHQLVLETTEAPKTDSKLVGIDQDYTYTFLFSRFIDGVRVENDNIHVGVDKRTGEVQ
NSYVNISKLAFPAKKPAVLDKDKAKELLLSQYQLRLVYQSVGLEGQIPPGIPVERYNVLAASGKYPTDNQNQKAQLVYTL
ENKYENVAYFLDAESGKWRSRDNGEVISLEPVKVDDIAGHWAQNELQLMLDYQALEVKDGKVKPDQSIKRGELIKMLVIA
DSGGGYYPAGAMDGRKNSFKDVEKSNAYFSYVETALDRKWIDRTSDEFNPDQPITREEMASLIVRALNYKKLTEFDNVFN
RNFTDASELKNIGNAAIVTGMGIMSADGGKFSPKLEVTRAQAATAFYRYLQKR
>lcl|NZ_CP026520.1_prot_WP_042231237.1_30 [gene=uvsE] [locus_tag=PC41400_RS00225] [protein=UV DNA damage repair endonuclease UvsE] [protein_id=WP_042231237.1] [location=complement(45414..46361)] [gbkey=CDS]
MEVRIGYVAQALAIYHNTPSSTFTYKRFKEGPYEESLAKAIEVGRRNLEATRRILYYNAAHGIRLYRLSSSLIPLATHPD
VHIGVKEVYREELAELGAFARRHDIRLSMHPNQFTLLNGSDAVVDAAIRDLDYHADILGGMGMGSDGIINIHVGGAYGNK
RASAERLLSNLPRVPEPILKRLTFENDDKTYTAEETLSVCQRSGVPMMFDLHHDWCNSSSRPGIELLPQIAETWKDRPMK
VHVSSPKNDKDFRSHAEYVEAGPLLVFLKACKEEGLDRLDVMVEAKRKDYACLKLAEDLSSVRGIRRLDGGVLSM
>lcl|NZ_CP026520.1_prot_WP_042231238.1_31 [locus_tag=PC41400_RS00230] [protein=FMN-dependent NADH-azoreductase] [protein_id=WP_042231238.1] [location=46578..47213] [gbkey=CDS]
MSTLLYITAHPHDDRASYSMAVGKAFLDAYRENYPGDHIVTLDLYKAYIPQIDTDVFGGWGKLRSGSSFDQLSEEEKKKV
TRFGELTDQFVAGDKYVFVTPMWNFSFPPVMKAYIDTICTAGKTFKYTENGPVGLLTGKKALHVYACGGVYSEGPGAGSD
FGSPYLKTIMSFFGISSFETLYVEGMAQFPDRAEQIKQDAIKHAQEKARTF
>lcl|NZ_CP026520.1_prot_WP_009675810.1_32 [locus_tag=PC41400_RS00235] [protein=hypothetical protein] [protein_id=WP_009675810.1] [location=complement(47341..47646)] [gbkey=CDS]
MSSETKRVLNVIQLIVEIGIIIGYVVGLIPFGFLWSGGWVVPLVFVSAVIGLINSNRTLLPAVVNIVLAFLSYIPLVGYV
TRIVGLLVSAYNISLIRRDQY
>lcl|NZ_CP026520.1_prot_WP_042231239.1_33 [locus_tag=PC41400_RS00240] [protein=ABC transporter ATP-binding protein] [protein_id=WP_042231239.1] [location=complement(48068..50155)] [gbkey=CDS]
MHTGKRLVQYALHHKKLFILALLMLTVAVAAELSGPFIAKRMIDTHILGIEQPWYKVPQETGSTGNTVDNGDTVSYGGDL
YKRGDRFAEGEARGDEARLLQVGTRFVFVNGPIAFDGQRSWQDGKLTVTRGGQSAAYEAKSLTKGELYVFYRPELAKLIR
LALLYLGLLAVAAVFTFGERYMLQVSANRVIQKLRLDVYRQIQRLPIHYFDNLPAGKVVSRVTNDTEAVRDLFVTVLANF
FTGFVYMTGIFIALFLLDVKLGAICLLIVPLLVVWVIIYRKYASRYNKIIRSRLSDINGMINEAIQGMTIIRAFRRQKET
MEEFETLNEDYLKHQQKLQSLNALTSHNLVGVLRNISFALLLWHFGGASMGTGAESVISFGVLYAFIDYLNRLFQPITGM
VNQLATLDQSIISAGRVFELMDERGEDVAEGRLPRYKGDVSFEQVSFAYKEDYVLKNISFEAKQGQTVALVGHTGSGKSS
IMNLLFRFYDPQKGTIRVDGQDITAMPKQLLRQHMGIVLQDPFLFTGTIASNVSLGDGTIPREKVDAALRDVGGERIFAN
LPGGFDEPVLEKGSTLSAGQRQLISFARALAFDPAILILDEATANIDTETEAVIQAALDVVKKGRTTFIIAHRLSTIRSA
DLILVLHRGEIVERGNHEELLELGGRYSQMYRLQQGGSAAGAAPGGVTTGVAAST
>lcl|NZ_CP026520.1_prot_WP_042231240.1_34 [locus_tag=PC41400_RS00245] [protein=ABC transporter transmembrane domain-containing protein] [protein_id=WP_042231240.1] [location=complement(50162..51913)] [gbkey=CDS]
MFSILQKLGWFFKKHWIRYTVAITLLIVAGIVEIIPPKILGYAVDEIHMGSITAGRLISMISWTLGITVIIYLMTYVWMY
QLFGGAFLVERVLRSRFMRHLLKMTPTFFEKYRTGDLMARATNDLRSVSATAGFGILTLVDSSAFMLVILFTMGVTISWK
LTFAALLPLPFIALAMKFYGKTIHERYTTAQDAFGQMNDQVLEATAGVRVLRAYVQEANEQERFRELTEDVYNKNIAVAK
IDALFDPTIKLLVGASYLIGLGYGAYLVFQTELTLGQLLSFNIYLGMLIWPMFAIGELINIMQRGNASLDRVNETLGYRP
DVQDVPQPAADAKPGTIGFREAAFKYPSSGTPNLEDIGFTLRQGETLGVVGRTGSGKSTLLKQLLRQYPTGSGSITIDGV
PIQQIPLEQLQSWIGYVPQEQILFSKTVRENIKFGKSDASEDDVLQAIERAAFTQDIGTLADGLDTLVGEKGVALSGGQK
QRVSLARALIADPEILLLDDAMSAVDARTEARIIENIRRERAGKTTLISTHRLSAIEHADRILVLDEGRIVEAGTHEELL
ALGGWYKEQYERQQVESNLTEAG
>lcl|NZ_CP026520.1_prot_WP_042231241.1_35 [locus_tag=PC41400_RS00250] [protein=cytochrome d ubiquinol oxidase subunit II] [protein_id=WP_042231241.1] [location=complement(52446..53471)] [gbkey=CDS]
MSDVTLAVSVLWVFLFTYSILGSIDFGAGFWSMYFSGQHHTKAGALANRYLSPTWEVTNVFLVLFVVALAGFFPKATHML
GAALLVPVSLVLLLLLVRSTFMVYAYSVSKHGRLLRVVSGITGLLIPGLLVSVLPITLGGFVTVENGVPVLHLGELFTSL
TEYAHLGFGLSTELFMSALFLADYSREAGDEETYRDYRRLAIIWGPINLLMAMLTVSTFSEHAPWMVEGFKRYAFWFGLS
VAAFIVAYILLLKRRPDGRRGYPRASIITVIIQYGLASYAYFSAHMPYMVYPHLTFDEGITNPTMFRSLFIGYIVSSAIL
VPVFYLFWRLFLKDKRYLQKE
>lcl|NZ_CP026520.1_prot_WP_042231242.1_36 [locus_tag=PC41400_RS00255] [protein=cytochrome ubiquinol oxidase subunit I] [protein_id=WP_042231242.1] [location=complement(53468..54838)] [gbkey=CDS]
MDTVVMARALFGTSMAFHIIFATLGVGLPMMIVFAEIMYHVKKDRDYVVLARRWTKASGILLGVAIPSGTIVGVMLSLLW
PGFMRIVGQVIALPFQIEMWAFFFEALFMSIYLYAADRLPPVVRIISVIFVAIGASASAVMITDAHAFMNTPRGFSINAN
GDFVDIRPWEAVFNPSFWSTALHVLSTAYMTGAFAVASVAAYKLLKARKAPREADYHRKGLFLALIFGGIMTAFTAVNGH
ATAQMLHHHIPEKLAAAEGLFETQSHAPLAIFGKVDEATQSIKGAIELPGFLSLLAGNRFDTVVRGLNEFPRENWPPLYI
HTLFNLMVFIGMFLLAVAFLSLLIRIWFKKPYPSWLLAILVASGPLSMLGIEIGWIFSCTGRQPWTVYHVQRTSEAALQS
QGIGTLFVMFVSVYVFLLVVTSIVMHVYFKRHPVSEALGLAAANGDQTGAAEGAST
>lcl|NZ_CP026520.1_prot_WP_042231243.1_37 [locus_tag=PC41400_RS00260] [protein=M56 family metallopeptidase] [protein_id=WP_042231243.1] [location=complement(55380..56885)] [gbkey=CDS]
MLLNFQPVFLFILNISFKAGVMVVLILTIKVLFKDRLPASWNYYLWLLLILRLLLPDSLGSQLSIFNYLDINPHEAVTLQ
PSSLQPNTTMTISNNNMDVNINGENVNYLKSFDSHSRKSNFYLLNWLSFIWIAGVLLISIAIYRANRKFAITLKKNSKPC
MDSSLLEQLETCKQIIGINRRVPLMTTSAVCVPSLYGLWRPRILIPPKGVESFSTEEWKHVFLHELAHLKRKDIGVNWIV
NILVIIHWFNPLLWLALRKMREDQEIASDYLAVSSMNKQQSQEYAAVLIKWMEYFQKPPLTVHSVSMLDNKRYLKRRIQM
IIQKRASYKWSLVGMLALALLAAVSLTSAKAEEEVKSEDVRVKEVVTKTFEALQHFQDADTFLSHTIDVRYSQPVANKLA
ADVFANKGLNPNTLIDFKIEKIEKLNVGEYKAYLSKTYSDLGKLPVYGIPVVKRDGEWLVVIENIEFVPSQFWLKNKDSI
TNSGKVEVLAVNENAAVIRRK
>lcl|NZ_CP026520.1_prot_WP_042231244.1_38 [locus_tag=PC41400_RS00265] [protein=BlaI/MecI/CopY family transcriptional regulator] [protein_id=WP_042231244.1] [location=complement(56887..57264)] [gbkey=CDS]
MNPNLRISDAEWEIMKVLWSAKKSLTASEIIESMSHPEWSSKTVRTLIARLVQKNAIKADMSSRIYQYYALYEEKDCILA
ETQSFLHKVRGNLFKPVIVEFLNKRSLSKEDIEELKSLLDKRSGD
>lcl|NZ_CP026520.1_prot_WP_042231246.1_39 [locus_tag=PC41400_RS00270] [protein=hypothetical protein] [protein_id=WP_042231246.1] [location=complement(57425..58327)] [gbkey=CDS]
MKKSGSMVLVLALAFGVFSSVTSAKVDDGLTEVEATVHKFYKSIKNYSDYDTYLAVSKDLRYSEEGRKQIAGSTFSNPKN
ANTLTDYNVEKLSKVSESEYRALVSQKFSSTGQVPAYEVPVIKENGKWYVIIEPIDFIKNDGLQSKNAPSANSSAKEIIA
KDDFVTVVKHKGEVVKNSSNNSVISPMASGVLYYEAKPINGSGYKPGTTTFSVAYNSGTFVKGWQLSRDSSVKAWIGYYL
KEYSPSGNTPVSNITIWNDYKQTADWYNVNFNNVPTSKPLVLDLENLTGYSTDLAGNVYN
>lcl|NZ_CP026520.1_prot_WP_042231247.1_40 [locus_tag=PC41400_RS00275] [protein=sigma-70 family RNA polymerase sigma factor] [protein_id=WP_042231247.1] [location=58524..59078] [gbkey=CDS]
MKTLNHSIFPLDFRNLNPDSQRSIINAFYELAYQNAILILKDRSLAEDAAHDTLIKVLECGPNLKHDHNIKSWIRQVARN
AAYDIIRKNKKYDYSLRHEKVINNLHSQSTEEFVEILLRNECLLKNIDELRDDYRVLIKMRYMDEMPYKIIAGKLNISES
VLSKRVERAKKQLISLFIHDWEPL
>lcl|NZ_CP026520.1_prot_WP_042231250.1_41 [locus_tag=PC41400_RS00280] [protein=hypothetical protein] [protein_id=WP_042231250.1] [location=complement(59781..60041)] [gbkey=CDS]
MTERDSSLQSQNPISQAQNSVEKVHNAVSQALSHPTAQLEEQAERAIQRAERAVENASLSFNQPAFEEVENGLRQEIEDL
HNGTRS
>lcl|NZ_CP026520.1_prot_WP_042231251.1_42 [locus_tag=PC41400_RS00285] [protein=sodium-dependent transporter] [protein_id=WP_042231251.1] [location=complement(60132..61499)] [gbkey=CDS]
MSKQDLNGGASGPAAGEKSGERFSSLGFIMSAIGSAIGLGNLWKFPYITGIYGGAAFVLLFIICLIIVGMPVLLAEMAIG
RAGRGNASTSFLNLSGSKRWSFFGFLGVFGSFMITSFYSVIAGWTLHYALSSFTGALYKGTDYAVQFQTFTSGYSPIAWQ
FLILGLSGWVIARGVSGGIEKFNKVLIPALAVVLLILMARALTLPGAGAGIEFFLKPDFSKLTGESALVALGQAFFSMSV
GMGVMITYGSYVKKEQSLGKATVAVTGGNLLYALIAGLIVFPTTFSFGLEPAQGPGLVFVVLPAAFSSMPLGAAFGGLFF
ILLAIAAFTSILSLLEVPVAFAMDRWKWSRLKATSWTTAVCFIVGVPSALSVGGVLSNFKIGGKTVFDSLDFVTSNILLP
IGGLVVTLFIAYASKRAMEEAGLKGTLLRVWVFLLRIVIPPLIVLIFLYSVGIIS
>lcl|NZ_CP026520.1_prot_WP_042231252.1_43 [locus_tag=PC41400_RS00290] [protein=ABC transporter ATP-binding protein] [protein_id=WP_042231252.1] [location=complement(61691..62572)] [gbkey=CDS]
MYGIETKSLTLSYNQTDIIEDLDLEIPKGQITVLIGCNGCGKSTLLRSMARLMKPKNGAVLLNGQEIAKQSTKEVARQMA
ILPQGPTAPEGMTVFELVKQGRYPHQNWLKQWTREDEDAVTKALEATGMAPLAERQVDSLSGGQRQRAWIAMTLAQGTNT
ILLDEPTTYLDMTHQIEILDLLWELNEKENTTIVMVLHDLNLACRYAHHIVAICDKKVYAAGKPEEIVTEELVRHVFGLN
CRLVRDPIFGTPTCVPFGKGRRLPGCEEMVNFVNAKSASSPDSPLSEPTPLPV
>lcl|NZ_CP026520.1_prot_WP_042231253.1_44 [locus_tag=PC41400_RS00295] [protein=iron ABC transporter permease] [protein_id=WP_042231253.1] [location=complement(62755..63810)] [gbkey=CDS]
MSRYKHIRLKWPSISFVLDKRALLVTAILAVLLCLLMLVSTGTGEIKISPLQVLRTLFGMGTPQDALVIHTFRLPRVLVA
VMVGAALAMAGAILQGVSRNPLASPDIIGITDGAGVFVVAFIVMTTGTGSTFLSQAQWLPLAAFVGASLTALIIMALAWK
NGLPPVRLVLIGIGIAASLSAVNTMLILLAPPVLTTQAVTWKVGSVYGSNWSHVGALALWLAVLIPILAYMARSLNVQAL
GDSVATSVGSRVNRQRVVLILLSTALAGSAVAFGGGITFVGLIAPHMARRLVGSAYGALLPASGLFGALVVLVSDYIGRT
AFPPHDYPAGIFTACVGAPYFIYLLYKSRNT
>lcl|NZ_CP026520.1_prot_WP_042231254.1_45 [locus_tag=PC41400_RS00300] [protein=iron ABC transporter permease] [protein_id=WP_042231254.1] [location=complement(63807..64814)] [gbkey=CDS]
MPLRTNTLKTIGLLTGAALLIIAMGCSIVYGYTGTSWHTAWDAYRAFNGSNEHLIIREARMPRALIAAAVGSSLAIAGVL
MQALTKNPLASPGLLGINAGAGLLIVIAMSLLSVSSLSTLAWIAFLGAAIASAVVYMLGSVGREGLTPMKLTLAGAAVAA
LANSLTQGVLLSNETRIQDVLYWLAGSVEGRKLDILLPVLPYLGAAWLVALLLSSQINVLMMGEDVAVGLGQRTVYIKLI
AAVVVVFLAGGSVAVAGPIGFVGIIVPHLVRSLVGFDHRWVIPYAGICGALLLVAADTLSRFIIMPQEVPVGVMTAFIGT
PFFVYIARRGFKTSS
>lcl|NZ_CP026520.1_prot_WP_042231257.1_46 [locus_tag=PC41400_RS00305] [protein=ROK family protein] [protein_id=WP_042231257.1] [location=complement(65041..65928)] [gbkey=CDS]
MRTLGAIEAGGTKFVCAIGNEDGTVLERASFPTTTPEETMGSVRAFFEGKGIEALGVGSFGPVDPNKKSPTYGYITTTPK
PHWGNYNILGELKKHFDVPMEFDTDVNGAALGEATWGAAKGLESCLYITVGTGIGAGAVVGGKLVHGLTHPEMGHIFVRR
NAEDTYVGKCPYHKDCLEGLAAGPAIEARWGVKGTELGTDHKAWELEADYLAQALMSYILILSPEKIIMGGGVMKQQQLF
PLIHVRLQELLNGYVQKEEVMAGIADYVVYPQLGDNAGICGALALAARALGSEIR
>lcl|NZ_CP026520.1_prot_WP_042231258.1_47 [locus_tag=PC41400_RS00310] [protein=S1C family serine protease] [protein_id=WP_042231258.1] [location=complement(66062..68554)] [gbkey=CDS]
MLNRNTFRLFGIVLAIVLAAASGFGFGTPVRAAADSDELPQIIARTSPSVVAIIGKPTGGKADKGTDKDRYDLAHGTGVI
IKSNGVIVTNAHVVKSMSNLVIVTSDGKSHTGKVTNLDEQSDLALVKIEAAGLQTASFADPKSLRVGETVVAIGTPVSFA
LRNSVTVGIVSGIDRSVNDQYRLIQTDAAINPGNSGGALVNMKGEVVGINTLKYSEYGVDNLGFAIPVDTVDYVLKHFEA
YGKVKRSFLGLKLEESWEAAVGLPSDDGLTVSHVEPGSPAEKAGIEEGDTLVSIGGKAVDTTVQYNEVLKQYLPGQSVKL
TVQAGGKTVTREVALGEEKSEDARQTDNTDNTGLNTDRGKTKIGDSSQSWSMKYPSGLIKQFQSADGTVVRFADAKGDFT
LSVSVEDQGDELSPALLLKTVGELKMGTILEKQYVKSGTQPYAKVTGKLGGMVYQIRLYQDHDKLYTLALTYSQDKENGT
KAGPNSYGDLLDSFRTVFDRQDAKLKDIAKKKSGGGDEVSNEYGLTFALPDKWNESPYRDDVQYSNDTMSQGISIEVTSQ
TTGDTLDAWVKRDKTDFEETFAPSYRETSEVKQGTLDGMESREVTFSATMGDKWESIHRVYLVKEGYKYKFSIHYPKELK
ADETDKLLGTVISSIRLDVKKMDAGVGFIQDEQEILDKNRTIRYVDEDYGYTVKIPETWTRDYAEDYDDSQFEKGGASFY
TFTGGYFAVDAEEGTDWQEKAAELNDGHEKSAEADADYKVKAEDVSLLGTTVRKFTVHYSVKKVGYEETEYVLSRNGKVY
VITTRLNDAVHTEANAERLNKAVRSMVFGK
>lcl|NZ_CP026520.1_prot_WP_042231259.1_48 [locus_tag=PC41400_RS00315] [protein=LysM peptidoglycan-binding domain-containing protein] [protein_id=WP_042231259.1] [location=68753..69130] [gbkey=CDS]
MITYVVQPGDTLYLIANRYNTTVEALLRANNLSTPYVYVGQTLYIPAPGPVPGPEPWPGPGPGPGPGPAPGPGPGPGFGN
LEQRVNRLEREFNQLNRELDRLTREVNRLDQRLDRLEQRVTRLER
>lcl|NZ_CP026520.1_prot_WP_042231260.1_49 [locus_tag=PC41400_RS00320] [protein=S-layer homology domain-containing protein] [protein_id=WP_042231260.1] [location=complement(69202..73455)] [gbkey=CDS]
MRKKRKIPQVLSVLMAGALAASAGTAGGSAVSAAPATQKAGFADLSGITGNPLESIQRAVELGLFDGAADAFRPKDPLTR
VEMAVLLTHALELPLDASSSSFKDVSASGWGSPYIEAIRKAGIVQGDGEGNYRPQAPVTRQEMAAMLMRAAELSPEGGSV
SVLPGDWDRTAPWAQSYVRASLEQGSMKLAGGSFLPAGTVSRGEAAEMLLATFFSVEHDMILQEIRDSRHIKLNGVVYEL
SEETAALFTEKNKAVLSRAKLKFHANGRKIDAIRSLELRASGSPAGDGEEEFSRNLVLDGNGAILPGDLIVSNDYLSVTG
LQINGDLTVAKGLENDFYAEGIKVKGQTRVAGGDDNTVVFENSELNGVTVLKENVRIVVSGKTVVQQMDIRSSAQVEVGV
EARVPELTLSDGSAKVDLHGSLGQVTVQSSQPAQITGSASIRQLNVTGSASVSIDGTGTIEQLQVTNTASQVAIGPSAQV
TSVSLGQGVSPSAVTTSSAPAGGAGSSSSSGSSNGAPVQTVPFPDYIFMERGTPASIDLAAYFTDPEGDTLKMSAISSSN
RIAGVSMSGYKVTLTPAASGTATIKVSIDDQHGNKISASIKVTVYALPVAQTVPDQLVQLNGASGVVRLSDYFSDKALRN
LTYSVSSSDLSVADVSVNGDELSLTPLRSGHTSVTVTAQNDIVAHDGSRGETSVTFELTVNTPPAASQNPADVHLAPGAV
SSPLDLSQLFTDGDGDTLTYEAESGDSSVATGSVQTGLLTIQANQPGTAVIKLRAKDGKGGVGTADLTVHVGPFNQSPVV
SLQPTDRLLAVGQTDTALDLTQIFADPDQDALVYEYTVSDPSVVAAVRQGDSLLLTALQGGQAVIKLTAKDGKGGEVSAS
YTVTVNRAPEAVRNPDDVYLVTGRADGELDLSQFFADGDQDPLTYEVQSSVPGTASWEVAGQKLKLHALQVGTSAFRVTA
KDGRGGEAAVTFTVNVRMPNTLGTIPDQTVELGAPKLLDLTALLAGFDSAATVTANTYDTSVATVLLSDRKLTMTPVSEG
TATVTLSVYHPVEGDQSTQFAIRVVQHLNSAPTVVKNIPDQLLTPGVTNVRTFDLASYFADADGDALSYTVSGFSPAVVS
ASVSGSILTLNPGTGSTDGQVTVTADDGKGGTAEFKINVRTAQLVNNEIVTIRTKQGIKDPLTYDLADLFPGETSFRMYK
GTPDAIISGPTVLNGKTLSLANETSYLWVIGAQGKAAVFRVVAEAQGAPQIFFSEYLDGGDGRIAFELYYSGDMDPGKKP
TGYTVEAHQYMKKTGKMNVISKALLDQGPGKPLIVPYIFIDSIFYDFFDLTAAVYYNEDGIRFYNPAEWNTVALVLKKDG
QIVDVLGDPSSTAQFLPNGGTLVRKSGVSTGSKQFSKYGEWNVYPKGSYQFMGHHTP
>lcl|NZ_CP026520.1_prot_WP_042231262.1_50 [locus_tag=PC41400_RS00325] [protein=DUF72 domain-containing protein] [protein_id=WP_042231262.1] [location=complement(73684..74535)] [gbkey=CDS]
MIRIGLAGWGDHDDLYTTPADKKDKLKVYSSHFPVVELDSSFYAVQPLRNTRKWVKETPPGFGFVVKAYQGMTGHLRGKN
PFESTEQMFRAFAESIEPMREAGKLTAVLMQYPPWFDCIRENVDALREAKSLLDGLPLAVEFRHRSWYAPEVRERTLGFL
EQEGWIHTVCDEPQVGSGSVPIVLRAANPDLTLVRFHGRNDGGWVGAPPGKSWRDVRYLYRYSRAELLEWKERLKVLERS
SREVAVIFNNNSGGDAAANAKEMMELLGIEDEQGLAPRQLDLF
>lcl|NZ_CP026520.1_prot_WP_084706632.1_51 [locus_tag=PC41400_RS00330] [protein=hypothetical protein] [protein_id=WP_084706632.1] [location=complement(74736..74954)] [gbkey=CDS]
MSHHHHHHHNHCLCPEKQVFDPPTRVYRDFYHPQIVEIVHPIEIVNRHHCVPVPKHVYCCTVRDEFCPSPIR
>lcl|NZ_CP026520.1_prot_WP_042231263.1_52 [locus_tag=PC41400_RS00335] [protein=DUF445 domain-containing protein] [protein_id=WP_042231263.1] [location=complement(75186..76466)] [gbkey=CDS]
MQELQSREKKPKETKHIATVSLGVMGAGFLATLPFAPHTAGAFLQSGFEAGLVGGLADWFAVTALFRHPLGIKIPHTALL
PNNRAQMTKAVVSAVEDNLLGKESIALRIGQMRLVETMLVKAEEALRSEGAREAVRSACRFALERLPAEQIAAFASQQIR
QAAGKMDPAAAVGLAVDQVLSHGYEEPAIDFLIDRAERWAVRAETRRLLGEIAEGVIQGIEMNGLMAFAKNALLGFMSGD
KLGGMLQHALLDNLAALRLPGHRNRVGLLDIVRREMAGLKDNERLHEAVGNLARQLPDRLRLEERLTAMIESWRDRLLEQ
LQEGPLLDETVMPLLEEGVRRLRGDENTVRKLEEWLKAQITEFVERNHSKIGALVRENIDKMDNATLIEFIENKVGKDLQ
WIRVNGAVCGFIIGLALEGVRMLFGQ
>lcl|NZ_CP026520.1_prot_WP_042231264.1_53 [locus_tag=PC41400_RS00340] [protein=metal-dependent hydrolase] [protein_id=WP_042231264.1] [location=complement(76680..77366)] [gbkey=CDS]
MQITYHGHSCFLVENEGYRVLLDPFLSGNALAVADPGEIKVNAILLTHGHSDHIADAEEIARKNDCVIIANFEIASYFAS
KGLKTFAMNIGGSAKFEWGTVKYTLAFHSSSIEVEPGKFVDGGQPGGILLTMGGKTFYHAGDTALFGDMKLYGEMYNIDV
AALPIGDVFTMGPDEAVLAATWLRAKKYIPTHYNTFDVIKQDASEWIQKLKQAYQEGVRLAPGETLDV
>lcl|NZ_CP026520.1_prot_WP_042231266.1_54 [locus_tag=PC41400_RS00345] [protein=putative sulfate exporter family transporter] [protein_id=WP_042231266.1] [location=complement(77685..78974)] [gbkey=CDS]
MKTFTQRLGTPALHETGKEKTDSNESQDKAESSVIKEETRPEEAQAEGRRGSADTFAASGFAEAEASNPSAAPAAAKLGP
GSGPARLATGLGMTLALALAAKALALIPGLNVIGQLVLALLLGIAWRAAAPVPASAPAGLAFASKKLLRAGIILLGMRLS
LLDVVHAGPAVFAASAGNILFATVVVYAIARRMNVGPRLAMLSACGTGICGAAAIAGISPQIKADDDETAVGVTVIALLG
TLFTVAYTLVYPLLGFSSAEYGFFSGATLHEIAHVIAAAAPGGSAAVDSALLAKLTRVALLAPVALVLGVWAARRERRGA
DEGAPGSAASGAAGSAAPRKAGVPVPWFIVGFLAAAALNTIGIVPQPAAAAAVQGAYLLIAMAMAGLGLNVNLAVFRRHG
LWASAAVLLGSVLLSLLGIAEVYVLRLLG
>lcl|NZ_CP026520.1_prot_WP_042231267.1_55 [locus_tag=PC41400_RS00350] [protein=LysR family transcriptional regulator] [protein_id=WP_042231267.1] [location=79247..80158] [gbkey=CDS]
MIVDHLRVFAAVAEQRHFSRAAEALNISQPGVSQHIRNLEQEFGAKLMIRSPKQVKLTEAGELLYARAKRILRLYEEARQ
EIDALQHVVTGSLKIGASFTIGEYILPRLLADYAKHYPKVDIQSTIANTVEVVQGVRRDHFHIGLIEGIADAPDVEIEPF
MEDEMILIAPRDHPLTRELFKLQALQNQTWVWREPGSGTRAYSDRLIAEWQLNVKRSFMFSSSQSVKEAVSAGLGIALLS
RWIVRKELESGEIVHLGAGGRRLTRTLTMVRRTAGENSKALEVFVQKVRGAHEAMQVPGPTPS
>lcl|NZ_CP026520.1_prot_WP_042231268.1_56 [locus_tag=PC41400_RS00355] [protein=phospholipase D family protein] [protein_id=WP_042231268.1] [location=80379..81962] [gbkey=CDS]
MEPLKRTEQKPSPPVRSSGNKRKPAEKSWLSSNWLKGGILLLVFFIWAGIVAAVHTHKDLPPGLSFEGKAYRAESVDFLY
DLSYKKDGGYHQEQMIFDRVNQAIAEAKEFIVVDMFLYSGYYDNDQSFPPLSGQLTESLLDAKRRHPAMPITVVSDEVNT
TYGSHPSPELERLKAAGIGVVLTDTDPLRDSVPLYSAVWRMGFQWFGQSGKGWLPNPLADSAPKVTLRSYLKLFNVKANH
RKVVATENTALVLSANPHDASFNNSNIGFQITGSKDFISEVVGTEQAASAMSGGPAHLQPKAADKQEEPARPEAASGNGT
QEQAAPSRDAVTVRLLTEGKTYARLLDSISAAPKGSEIRMGMFYLADRKVIDVLLDASARGVSIQLLLDPNENAFGHSKT
GIPNRPVAAELRKKTGGKIAVKWYNTAQEQFHTKLIDIRHPDGSAVINGGSANFTPRNLDDLNLETNVEITAPAGSKITQ
DLDRYWDRLWNNRDAEYSLEYPAYEDKLVGFKRIVYHLQDMLGFTTY
>lcl|NZ_CP026520.1_prot_WP_042231269.1_57 [locus_tag=PC41400_RS00360] [protein=permease] [protein_id=WP_042231269.1] [location=82149..83207] [gbkey=CDS]
MNTSPIPVSTKSRPRSAYWKPFLFIAILIGGLFYVKWEPYFHKALTAAAKHDIGKSILAGSAEPSLAAAWDYSVVYFKAV
WKAAILGIVLGSLLQVLIPQDWLKRVFGKRTFGSTVAGGLSAMPGMMCSCCAAPVAVGMRKQSGSIGSALAFWIGNPVLN
PATIIFMGFVLGWKFAVLRIAAGVVLVFGVSYWAERLVKKDDAVTGELPQIPAAQHQSGPVVVRWLKALWTMFWTVVPVY
FIAVFILGAARAWLFPALDPAIGNSLVALIGLALAGTLFVIPTAAEIPIAKTMMALGLGTGPAAALIMTLPAVSLPSLIM
VRKAFPARVLWFTAGMVALTGIVTGVVGIFVL
>lcl|NZ_CP026520.1_prot_WP_042231271.1_58 [locus_tag=PC41400_RS00365] [protein=energy-coupling factor ABC transporter permease] [protein_id=WP_042231271.1] [location=83883..84650] [gbkey=CDS]
MNIRKTLASFLLIAGIVCFFLVNEPESAYAMHIMEGFLPPFWALFWWVVFLPFFILGVRALIRITKETPELKLMLGLAGA
FTFVLSALKIPSVTGSSSHPTGTGLGAILFGPLPMSVLGSVVLLFQALLLAHGGLTTLGANAFSMAVAGPFVGYAVYKLM
IKSTGRQKLSVFTGAALADLTTYLVTSVQLAVAFPAADGGMLASFAKFAGIFAVTQIPLAISEGLLTVLVLGWLQAYTPH
EISALQRNLKGADRS
>lcl|NZ_CP026520.1_prot_WP_009675911.1_59 [locus_tag=PC41400_RS00370] [protein=energy-coupling factor ABC transporter substrate-binding protein] [protein_id=WP_009675911.1] [location=84647..84934] [gbkey=CDS]
MKTWKKNVWILVVVVLLAVLPLIFVKGEFGGADGAAEEAIQEIAPTYEPWFASLMEPPAETESMLFALQAALGAGFIGYV
IGLYKGKASKPKPKA
>lcl|NZ_CP026520.1_prot_WP_042231272.1_60 [gene=cbiQ] [locus_tag=PC41400_RS00375] [protein=cobalt ECF transporter T component CbiQ] [protein_id=WP_042231272.1] [location=84931..85740] [gbkey=CDS]
MMLKRIDTISYANAFRGISPVWKCGFAGCLFILSYAGAPPVQAAVFGWMTLWTLLYARVPGKLYWLVYGGSLLFFAASLP
ALLLEFGKNAGTGVPLSSGWIIGTAGPWTVYVTGESLSRTAGLFMRVSACLACLQFVLFTTPLLEILQVMKKIRMPQIVI
ELTLIMYRFIFLLADTAHAMRTAQRCRGGQNGFAGGLRDTALLIVHLFGRTMQRYRGLTYGLTSRGFTERIELAPYEAAP
VPPRYKAEGVTGIFMLTAAQIWLKWRGFA
>lcl|NZ_CP026520.1_prot_WP_042231274.1_61 [locus_tag=PC41400_RS00380] [protein=ABC transporter ATP-binding protein] [protein_id=WP_042231274.1] [location=85737..86591] [gbkey=CDS]
MNPIMELADVVYSYPGTGKKALAGLSLTIPAGRKIALCGHNGSGKSTFFLHAVGIHRPQSGTVRWKGAPVSYKRTDLQEL
RRQVGLVFQDPEQQLILGTPYEDISYGLRNVGVSEPEIRARTEAVLERMGLEQLAHTPIHHLSLGQKKRVTLAGVLVLEP
SLLLLDEPTAYLDRASERLLLDELADIRQSGVTVVMATHDMDLAYTWADEVLVMDQGACLLQGPPGRVFAETDTIERIGL
ELPMLLRLWRSLPRALREGVDPPRNADAFGELIAARLQAAAAQS
>lcl|NZ_CP026520.1_prot_WP_042231275.1_62 [locus_tag=PC41400_RS00385] [protein=glutamine--tRNA ligase/YqeY domain fusion protein] [protein_id=WP_042231275.1] [location=complement(86746..88428)] [gbkey=CDS]
MENKSATTNTTNTNFIKNIVIEDLNEGRVDTVVTRFPPEPNGYLHIGHAKSICLNFELADEFKGKTNLRFDDTNPAKEDE
EFVRAIKEDVQWLGFEWDGLFFASDYFDYMYEKAQLLIRKGKAYVCQLNADQIREGRGTLTEPGKESPYRDRPAEESLDL
FERMRKGEFKDGEMVLRAKINMASPNINLRDPVIYRISHAHHHRTGDAWCIYPMYAFAHPLEDAHEAVTHSICTLEFEDQ
RPLYDWIIQECETEHVPHQYEFAKLKLTDTIMGKRYLKQMVDEGVVDGWDDPRMPTVSGLRRRGFTPESIRNFCREIGVT
KSDSTVDVKYLDHFIREDLKLKAPRTMGILKPLKVVITNYPEGQVEMLDAEINSENPEMGIRQIPFSREIYIEQDDFMEN
PPSKYFRLFPGNEVRLKHAYFIKCEEVVKDEAGRVIELRCTYDPETKSGSGFTGRKVKGTIHWVEATRAVPVEFRLYEPL
VLPEQEEKEEGSTFLDRINPNSLEVLSGFVEENMKDAEGNDKFQFFRHGYFNVDPKASTKDHLVFNLTVSLKSSFDIKQA
>lcl|NZ_CP026520.1_prot_WP_042231276.1_63 [locus_tag=PC41400_RS00390] [protein=S8 family serine peptidase] [protein_id=WP_042231276.1] [location=complement(88745..91972)] [gbkey=CDS]
MRKISLLVLAVLLVLSLSVPAFGAESPLSLKKSVSTPRYERPQAAKDALPGQVIVKYKEKADTMVMQGLMGPEKRAELPD
PPSSKYKQRVGSKGLVTMEFPKSEPVSDVLEELENDPDVDFVEPVYPVWAAETKAAIRNGGAGQGSAVRTTTGSASAADS
PAKAASLPNDPYVEQQWWYRAIQSDLADTSTLDAKLPGVVIAVVDTGVDSSHDDLKGSLVEGYNFVSNTPDTRDDNGHGT
NVAGIAAALKDNGVGIAGVASGAKIMPVKVLDSEGYGRTDVITEGIYWAVDHGADVVNLSLTSASSSRVMEEAVQYALQR
GVIVVSAAGNESNHWTGNDTGELDQVKGSEDPLAHRFVKDVAFPAKLPGVLAVGAVDWYPQTEFIVADFSNSGSSLSVVA
PGVDIYSTNKGNGYTGRNGTSMAAPMVSGLAALILAGDEGLDDLTTSEKVERVGRIISETAKDLGPPGFDDEFGSGLIDV
KKAMETPRISIQAANGSTNLTGTGELRADIRLLNKDGSLAADAGGAGKADLFTYTSNGTEPQFVKTLTFQASQGKASLTY
KPDKAGKYILVAYNDEKSPLWVSGELTFQMKPQAPQASLRSGSYTGTQKVALTASDPGAQMYYTLTGAAPTVQSAAYTGP
IEISRTQTLRVISAAGGAVSDVQTYRYEIVTSSGNTGGGDGNGKPSEETVTDVKADSTVIQEKGYAMLDVKVTDENLQPL
LKNPDVALIGVQAVSAGKNNTAGAKVHLPASFFASGQSKGLLITTSEGILRMEAGSLQVDGSEEVVISLRRTHATESLTS
AYPEGASPMSSILEIKITAGSRMIGNLQRPATVTLSHAAASDLGVDPEKMGAFVYSEDDSEWQYVKGTQNEGSMTFAVEG
SSYAAVFSVDRTFADMSTHWAKRPVEVLAARQIVDGMDANRFVPQAGVTRAQFAAMLARLLQLPAAPDGKQQFRDVTKGA
WYADVIARSYSAGLVSGVTDDRFMPDEPITREQMAAMLVRAFAYGSGIAPDELPVPKTASFADEGTAGAWAVSFMRKAAG
LGLIDGYEDGTFKPLNGATRAESASVIYKLSAREY
>lcl|NZ_CP026520.1_prot_WP_042231277.1_64 [locus_tag=PC41400_RS00395] [protein=bifunctional ADP-dependent NAD(P)H-hydrate dehydratase/NAD(P)H-hydrate epimerase] [protein_id=WP_042231277.1] [location=complement(92367..93935)] [gbkey=CDS]
MRIVSGEDMYEIDRQTVNLIGISADTLMESAGRAAASLLLERIPPEAKVVVLSGAGSNGGDGIVIARVLKAHGRRVSLWL
IPPRDKIRGAAAAALSVYEKSGYAAIDYTGSEAELERRLERADAIVDALLGIGVNGPLREPYRSLIGKINEQHAAVYAID
LPSGVPANGGPVEAAVRADVTVTIQYPKLGAFTHPAASYCGELLVADIGIPPAAAAGLPRRSLWTLADVRRTLPRRKASS
HKGTHGRLLVAGGSLRMPGAVVLASRAALRSGAGLVTTAVPDCIHAIAAGHYPEAMYLPSPSEAGHFRGELPLGDAAFGA
IAAGPGLGRTPGSRAVVEQVLRQDVPVLLDADALFHAAELLPLLRKRRKPTVLTPHPGEMARLTGSTAAAVEDDRFGVSA
RFAAETGAYVVLKGPHTIVAAPDGMQRVNTTGSAALAKGGSGDVLTGIAAAFLLQHEDVLQAVSNAVYIHGLAASLLTEE
HHSLADVLASDVIESIPSAFRHIYGDEGGAEGKSPKPGDSFF
>lcl|NZ_CP026520.1_prot_WP_042231278.1_65 [locus_tag=PC41400_RS00400] [protein=MerR family transcriptional regulator] [protein_id=WP_042231278.1] [location=94181..95188] [gbkey=CDS]
MHIKEAAKRLGISARAIRFYEEKGLIAPGKELSSGYRSFTEQEIWRLRTIVALRESGMSVSGIQAALEQVERGEHDELLH
YLELQRSVLYARMLETRQTLQTTDHMIGLLQGSRGLPLEDFYRLAQDSREVRELRNWEDRWNFDELASSHDERVWSGDSA
YPGYREALERTVEEINPAPGETGLDLGTGTGNLAGLFMERGAVMNGVDQSKEMLRICRSKFPGMTAKIGNFLAVPYTDGQ
FDFVVTSFAFHHLSGRQQKLALNEMARVLKPQGRIAIAGFINRNADGGASADKHRTNSPTGALDFLDWFGPNGFESKCLK
LSPELYLAAARRGLP
>lcl|NZ_CP026520.1_prot_WP_042231279.1_66 [locus_tag=PC41400_RS00405] [protein=class I SAM-dependent methyltransferase] [protein_id=WP_042231279.1] [location=95580..96278] [gbkey=CDS]
MPDHTDIYRNQTHLYDRMISRQPDLAERIDRIRPHEGLDIVDLGAGTGRLTAVLAPKAASVTALDASSSMLEVTASKLRR
TGLTNWRCTAADHRSLPLADDSADLLVCGWSLCYLAGSDVPGWEAHVAQALAEMRRVLRPGGTLLLLETMGTGTEVPDPP
AFLKPYYALLTQKYCFAHETFRLDYTFESLKEAEETARFFFGEDLAAKTTRNNWVTLPECAGVWWLHMPAEA
>lcl|NZ_CP026520.1_prot_WP_042231282.1_67 [locus_tag=PC41400_RS00410] [protein=NAD(P)/FAD-dependent oxidoreductase] [protein_id=WP_042231282.1] [location=complement(96656..98095)] [gbkey=CDS]
MSNRYDVIIVGAGPAGIFAAYEFSLKSPHLKVLLVDKGHDIGSRRCPILEEKIKLCPPAAGRKEFAGCLPACSITAGFGG
AGAYSDGKFNITTEFGGWLTDYLSPSRVLDLIRYVDDINLKHGAQQTLTDPLTDVVSSIENRAYAAGLKLLRAQVRHLGT
EQNLEILRSINTYLKTKVDMRFKTEVADLLTSKTAEGHKIRGIVLSNGEEIEADSVIVAPGRDGSAWLTSVLRKRRLKMY
NNQVDVGVRVETSDVVMQEINKHLYEGKFIFNTSVGTRVRTFCSNPSGHVVVENHSGVMAANGHAYKDPALGSQNTNFAL
LVSHQFTEPFDKPNEYAREICKRANDLSNGGVIVQKYGDIIRGRRSTEGRIKEGFLEPTLKEAVPGDLGLVLPYNTMKSL
VEMVEALEKVTPGIASEHTLFYGVEAKFYSARPKLTEQLETEIAGLYCGGDGAGVTRGLAQAGAAGVWIARAVAERAQG
>lcl|NZ_CP026520.1_prot_WP_042231284.1_68 [locus_tag=PC41400_RS00415] [protein=LysE/ArgO family amino acid transporter] [protein_id=WP_042231284.1] [location=complement(98419..99048)] [gbkey=CDS]
MMLTAAAALHGLILAMGLILPLGVQNVFIFNQGALHKKYIRALPAVLTASLCDTLLIGLAVFGVSVLVLKIWWIKTVLMS
AGVIFLAYMGWTTWKSASRKDADQPAAGDETRYPARKQITFAASVSLLNPHAILDTVGVIGTSSLQYSGTHKWVFALACI
LVSWFWFFGLAALGWLTGRISGSARFTAVLGKVSAIVMWLSAGYIVFSF
>lcl|NZ_CP026520.1_prot_WP_042231287.1_69 [locus_tag=PC41400_RS00420] [protein=PLP-dependent aminotransferase family protein] [protein_id=WP_042231287.1] [location=99247..100698] [gbkey=CDS]
MQDSSWKPDKTSSIPLSVQIKQYIKSKIENGEWPVGTKIPPQRAMAAVFEVNRSTVVAAVDELIADGILEGRSGSGTRVV
NNSWTLLTPSAVPDWTSYVQSGVHRPNRTMIREINRAEFVPGMIRLGTGELSPALHPHEMTRKVLERLPDRMTSLGYEEP
KGLLPLRERISAYLSTSGIHASPSSILIVSGALQALQLISLGLAQPGSTLLLEKPSYLFSLHMFQSAGVRLCGLPMDERG
IMADRIPELAKRAKGAMLYTIPSFHNPTGTLMPEDRRKELLEVCETAQLPVIEDDVYRELWLDEPPPPPLKAMDKNGLVL
HLGSLSKTWSPGLRIGWVVGPEAVIDRLADLKMQTDYGSSSLSQWAAAELLSGGLYPRHLETVRTEMRRRRDRTLELLDS
GFKDIASWKVPSGGFYIWLKLTRSLSMKELFEKGLQAGILLNPGSLYDPLSAPYLRLSYSYASLEDLEKGLLKLSALIRR
LAR
>lcl|NZ_CP026520.1_prot_WP_042231290.1_70 [locus_tag=PC41400_RS00425] [protein=DUF3298 and DUF4163 domain-containing protein] [protein_id=WP_042231290.1] [location=100809..101438] [gbkey=CDS]
MASLFQLPVQVETETVTSPGVTIYTPKLSGLRSPAVQKKLNAAFSGLTALLFNEQKKYQTGTLQEMTGHYEVKTNERGLL
SLIATNYAYSAPMAHGYTVAESLTCDVNSGDFFTLSQLFKPGSGYVELLSQHVKVQLQRRKLPLLNPPFKSVWPDQNFYL
ADKALVLYFDLLDITPYYVGFPMFPISVYDLLPIAAEGGPLDILAVDVS
>lcl|NZ_CP026520.1_prot_WP_042231291.1_71 [locus_tag=PC41400_RS00430] [protein=group 1 truncated hemoglobin] [protein_id=WP_042231291.1] [location=101664..102020] [gbkey=CDS]
METSLYEKLGGKEAIGKVVEEFYKRITADESVNHFFRDTDMAKQLRHQTAFISYALGGAPYTGKSMAAAHTGLNLQPEHF
DSIVNHLQASLAHFGVTEDDIRQITEKVGSLKGDILYK
>lcl|NZ_CP026520.1_prot_WP_042231294.1_72 [gene=dgt] [locus_tag=PC41400_RS00435] [protein=dNTP triphosphohydrolase] [protein_id=WP_042231294.1] [location=102213..103529] [gbkey=CDS]
MDMRKYRLDEPALGTFSEERDEYERDYARLIQSPVFRRLQGKSQVFGAGTGDYYRTRLTHSLEVSQIAREVARRMRKQYA
FCIGGEHPGLMIDPAVVECASLAHDLGHPPFGHKGEEVLNHLLGEEHGLSYEGNAQNFRILTFLEKRAGSDSGLDLTAAV
LLAINKYPYSLEEPGRLKGVYGMEWEGIRQLRDMWNMPEGCPTLEAQLMDLCDDIAYSTHDIEDGIRAGKIQMNRSFFED
DRLIGHLVQEIVNDQGNLEVGWEQVDITAMVKQVLASYLEQWELIYAQCDREASRTRREMKARWVSSFAGKVGIIDDPDK
GWKRVTFVREGRRDLDLLRTMEIIKKLAWVTLVKDFRVQRLQKRSEIIIRRLWDSFKVPDHGQLIIPPDWIENYERHLSV
WSWPRLVADYISGMTDAYAEKVAAELYASKSGSIYEMD
>lcl|NZ_CP026520.1_prot_WP_042231296.1_73 [locus_tag=PC41400_RS00440] [protein=EAL domain-containing protein] [protein_id=WP_042231296.1] [location=complement(103568..104611)] [gbkey=CDS]
MQNFCSKCFVGERGYTVYFGGSETSAFLDAYLPEYPADKWDKIDSRLYWMHESLFNAMLDYVTAHLEYANIRAVRADKHD
PLQGLDKLKPVQFFKEERDANWIDDVITNRSIRTFYQPIVSFDGVESSIIGYELLSRGTGRDGGLIPPVQLFQAAKARDR
LFALDRTCRLEAVRNAGGLDDGLIFINFLPTSIYNPVHCLQSTVEMVERTGVNPESIIFEVVESEEINNMDHLRSILAFY
KKHGFRYALDDVGTGFNSLQVLSELEPDVVKLALEFCRGVSKDKRKQRVARAVARMTQELNSRALAEGIEDRDDLECLRD
LGYEWFQGYYFGKPEPQPLRVLPSWTA
>lcl|NZ_CP026520.1_prot_WP_042231299.1_74 [locus_tag=PC41400_RS00445] [protein=DUF4846 domain-containing protein] [protein_id=WP_042231299.1] [location=complement(104744..105682)] [gbkey=CDS]
MSFLYSDKWKQSALMSLLVLSLGVSACDSAGSSEPTAKQAGKEAGSGKSAAGQKAAQNAAILTPEGVTLQERFKAPKGFE
RTPAEPGSFAAFLRGLKLKPDGSVVHLFDGREKNKPGVYEAVVDIPIGSRDLHQCADAVMLLRAQYLFDRKQYGEIHFNF
VNGFKAEYSKWMNGYRIKMNGNEASWVKNGGSDNSQEGFRKFMDVVFAYAGTLSLEKELIPVKKSEVKPGDVFIVGGSPG
HAVIVVDTAVNKSTGETLFMLAQSYTPAQETHILVNPANPAQSPWYTLKEGGGLKTPEWEFGENSLRRFQGN
>lcl|NZ_CP026520.1_prot_WP_042231301.1_75 [locus_tag=PC41400_RS00450] [protein=dicarboxylate/amino acid:cation symporter] [protein_id=WP_042231301.1] [location=106095..107450] [gbkey=CDS]
MNNNLIKAFETNWISLVVSLVIIGFLYFLAKKRVSFGYRVLIGLGIGLIVGFIFNTYKIDAKSVSTIGTIYVNLIKMMVI
PLVVVLVINSIAGLSNLGQLRKIGLKTIGLFLATTGIAALIGLLVALVFNPGSGIPQGTLKDFKPKEIPTFSQVILDLVP
SNPFNEMATGKVVPVLIFAIFVAVAIVHIGSKKPDAVKPVKDLIASLAQVLHQVIKYVIRLTPYGVYALIAAMAAKYGVD
TLKPLIMLILATYTALLLHFIITFGGLVTFVAKVNPFKFFRKAYPTIAVAFTTRSSYATLPVNLEVLTKKLRVSPRIASF
VAPLGATINMNGCGGVWPAVVAIFVAGVYHIPLSVSDYILIVLVSMISSVGVAGVPGPAAISTTVVLTALGLPLEGLGLT
LAIDAIVDMGRTAVNATGTTVSTLIVANSEGEFDRASFDRDEEDELELNPA
>lcl|NZ_CP026520.1_prot_WP_042231303.1_76 [locus_tag=PC41400_RS00455] [protein=YhgE/Pip domain-containing protein] [protein_id=WP_042231303.1] [location=107727..109871] [gbkey=CDS]
MRKIGSIYARDIRNIATNWVAAVIILGLVFLPSLYAWFNIEASWDPYGQTGGIAIAVANNDEGATLRGKQLNLGNEIVAS
LKKNEKIGWTFVDEKEAMKGVNHGEYYASIIIPADFSAKIATVLTDDPQKADILYYVNEKINAVSPKITSKGATGVIDEV
SRNFIKVANGTIFQIFNEIGVELENELPTIEKVKTLVFKLETVFPELNQAVGVALDDVHKSNDIVKKAQDNLPIVTQLAK
DGEEFTNKLGDFLGKGSEAVEALGPNIKQDLLLLQQTATATQELTGILKDANFDPAVAKAAIDRVSQRLTVAVDVTDRMI
GLFEQLNKYTGGKVMPFVIDRLKSVKDRFQQQLTTVGKIRDAIDKGEKPAQELVDNLNRLSKETAGFIGDILGRYDSEIQ
PQILQGIAKAKKTAQNAHTVLSEAVKSIPDLQKILGDAAKGLAVGDKELKEIQARLPEAETKIKDLADKIRAFEKEGNLT
ELIDLLKNNFEKESEFFAEPVQLKENKLFPIPNYGSAMSPFFTTLSLWVGALLLVSLLTVEVHEPGYKSYQVFFGRYLTF
GTLAVMQSLMVTLGDMFILKTYVVDKLWFVLFGMVLSAVFMLIVYTLVSVFGNVGKAMAIVLLVLQLAGSGGTFPIQVTP
PFFQAIHPYLPFTYGISMMREAVGGILWDIIMEDLVMMAIYIGITLLIGLALKKVINKSSAKFISRAKESKLIH
>lcl|NZ_CP026520.1_prot_WP_042231305.1_77 [locus_tag=PC41400_RS00460] [protein=MarR family transcriptional regulator] [protein_id=WP_042231305.1] [location=110023..110469] [gbkey=CDS]
MDIYQSDSVMQILSRVFKLHRRHLHELLQNYEVYPGQPPLLIRLSERDGQSQHELAAQMRVKPATLTVMITRMEKTGLLR
RGADSKDQRVSRVYLTDKGHQATTAVKQALREIEETCLKGFLTEEKLLFRRLLLQMYENLKDIEEQID
>lcl|NZ_CP026520.1_prot_WP_042231846.1_78 [locus_tag=PC41400_RS00465] [protein=ABC transporter ATP-binding protein] [protein_id=WP_042231846.1] [location=110497..112236] [gbkey=CDS]
MLTLSRYLKPYWKAAVLAPLLMLLEVYMDLLQPKLMADIVNQGVAAGDLATIRNTGLLMLGVALIGLIGGAGCTVYSSIA
SQNFGADLRDDLFRKTQTFSFRNLDKFKSGSLITRLTGDIVQVQTLVQMTLRVLVRAPFLAIGGVIMSFMLNPRLALILA
AAIPVLFLVLYLLIRVSQPLFLKVQEKLDRVNTVLQENLSGIRVVKAFVRSDFERKRFGEANRDFTDIALKAARTIALNA
PIMTFIMNVSLVAVLWYGGAQTWNLSFPVGDLIAFLNYVTQVLTSLLTVGMMMMFVTRARASAGRINEVLEAEPEIADKP
GASADAIRAGRVVFDRVSFAYDEDGRELALRDISFTVEPGQTLAILGATGSGKTSLVHLIPRLYDPAAGRVLIDGTDVKD
IALEHLRSRVGMVLQQAILFSGTIRENISYGKPDATQEEIEAAARAAEIHSFIAGLPEGYDTLLGQRGVNLSGGQKQRIS
IARALLVRPTVLILDDSTSAVDLATEARIQQALKERMARSTCLLIAQRISSVVDADQILVLEDGRIAARGTHAELLAVSP
VYRDIYKSQQGREEVIHGA
>lcl|NZ_CP026520.1_prot_WP_053228744.1_79 [locus_tag=PC41400_RS00470] [protein=ABC transporter ATP-binding protein] [protein_id=WP_053228744.1] [location=112226..114109] [gbkey=CDS]
MAHKPQDNGSSAPEPPMGLGVGGPAGRGPGPGGPRMTQKVRAKNTGATLGRLSRYLGRQRKGLAAVVLLTVLSAGAGLIA
PYLLGRAVDDYIMPRDYGGLLQLCLLLLGIYLFSSVMTWFQTYVMAAVSQRTVQNLRHDIFAHLQKLPLNFFDSRTHGEL
MSRTTNDIENVSNTLNQSVIQLMTSVITIAGSLVLMLSMNIWLTLVTLLSVPVTIYLTGQISRRTRTHFKGQQKSLGELN
GFIQETISGQKVVQVLRRERQTAARFKEINAELREAGTKAQIYSGMVGPFNNVVNNFSFALIAAIGGWLVISGWTTLGVI
VSFLNYSKQFTRPINDLANQYNLIQSGIAGAERVFEVLDAKTEYPDIPQQTKGSSADARPLNVAGEVVFEDVSFSYKPGS
PILSDISFTAKPGDTIALVGPTGAGKTTIVNLLTRFYEIDEGEIRIDGTDIRELDKDSLRGQLGIVLQDAYLFSDTLREN
IRYGRLDATDEEVEAAARLANADGFIRKLPQGYGTLMSSEGGNLSHGQRQLITIARAILADPAVLVLDEATSSVDTRTEM
HIQEAMSLLMKGRTSFVIAHRLSTIRDADLILVLNGGKILERGTHEELLTAGGFYYELYSSQFRDAV
>lcl|NZ_CP026520.1_prot_WP_042231307.1_80 [locus_tag=PC41400_RS00475] [protein=hemolysin family protein] [protein_id=WP_042231307.1] [location=complement(114360..115661)] [gbkey=CDS]
MESDSYLLNLFMIALLIGLSAFFVAVEFAVVRLRGSRVDQLIAEGKSNARAVKQVLANLDGYLSACQLGITITSLGLGWL
GEPTVEKILHPVFESLQLHESLTGLLSFLIAFIIITFLHVVIGELAPKTIAIRKTEAVALLTARPIIWFNKVMYPFIWVL
NGSANQLVKLFGIKPATEHEEAHSEEELQIIINESFESGKINQAEYGYVSRIFAFDNMLAKDIMVPRTDMICLYANKSRE
ENLAIIKEQQYTRFPVVEESKDNVIGIVNTKKFFLAYEDDEELDVTTLIQPVMAVSEVIPVNALLKRMQKEGTHIAILID
EYGGTSGMVTIEDILEEIVGEIRDEFDMKEEPEILVKSEGHLVLDGKVSVPIVNDRLLVELETERWDTIGGWLYGHRDDL
SEGESLTFDNVKFTVLEREKNRYLKLEVMRIDD
>lcl|NZ_CP026520.1_prot_WP_042231309.1_81 [locus_tag=PC41400_RS00480] [protein=hypothetical protein] [protein_id=WP_042231309.1] [location=115870..116454] [gbkey=CDS]
MKKQVFMLTSTLLLLAGCDNMGTGEKIEAAGTPVSTAVTPSSGTGKTSATVAPSSSTAGTTAPVSTPVQLPTVSPVASNP
AKPPTQEEKKTSVRLADQSGTAAGQGSFLFDFEAGDGFQFYLENKGTQRLFYRVNYPNGQNQVEKTILQPGESHSFTYQK
SSPDLPVHGVYTVHMYNDDGSSIDYALVVRNLNL
>lcl|NZ_CP026520.1_prot_WP_042231310.1_82 [locus_tag=PC41400_RS00485] [protein=MMPL family transporter] [protein_id=WP_042231310.1] [location=complement(116511..119696)] [gbkey=CDS]
MRTVLKLRWLVLVLWIAVAAGLMLTAPNMESLVREKGQITVPEGYSSTKAAKLIEEMKKGSPEAEGERSSVLVFHGTEKL
TQAQLDEIKRGVDKLKNAKDSAGVSSVMTHFDNKELEKQMVSEDGKTVLVLVNTAKDGREAKEVRDALYGTVSDVKVEHY
YTGNWLIQEDVMASSQEGLKKTEYITVIFILAILFIVFRSAVAPFIPLLTVGFSYLVSQSVVAFLVKYADFPLSNFTQIF
LVAVLFGIGTDYCILLISRFKEELAHSGDKTEAILATYRTAGRTVFFSGLAVLVGFASIGLSTFVLYRSAVAVAVGVAVM
LLALVTLVPFFMAVLGKVIFWPAKGSLEHKPSRLWGAVGSFSLKKPLWSLAILAVIIVPFLVAYKGSTSFNSLDEIGEKY
DSVKAFNLISSSFGPGDTLPSTVVVKTDKPLDNSEGLAAVEQITRELAKVDGVKNVRSVTRPTGEPLDDLQVSTQVDKLE
GGIGQSGDGLGEIGKGLSEASGALSANAPKLNEAANGAKQLVDGTNALKTGVVQLGDGLERLEAGLRSGSAGAGELRAGL
AQAKASADQLAAASNDLLAGYKQVGGGLGQLTQAYGDVASQASGLAQGLGDVGKGLGGLAQKYPELKSDPDFLKTQGAVT
QLQTGAAQLGEGLKQLNQQLAGANQGLQQANAGFEKAAAGQAALAQGLAQLSDGIAKLQAGIDQAADGQGQIVAKLPELT
KGFDKLNSGQKELQQGFAQLNGQLGELTGGLDKSVNGLSQVSGGLKSAQDYLKELSAAPNKQISGWFMPQDVISGTEFQP
SLDAYMSKDRKIAKFEVIFEGNPYEVETLEQSGELTAAVERAVKGTPLQNAAYAVDGVTSMNNDLKNISAADYSRTAMLM
LIGIALILIVLFRSIVIPIYLILSLLLTYYTSMAITEVIFVRMLGYSGISWAVPFFGFVMLMALGIDYSIFLMDRFKEYR
HLSPQQAILEAMKNMGTVIMSAALILGGTFAAMLPSGVMSLLQIATIVLCGLFLYALIMLPLFIPVMVRMFGPANWWPFM
GRKERGEDASYSADSASSSLH
>lcl|NZ_CP026520.1_prot_WP_042231314.1_83 [locus_tag=PC41400_RS00490] [protein=MarR family transcriptional regulator] [protein_id=WP_042231314.1] [location=complement(119693..120160)] [gbkey=CDS]
MDETLLSQIIDRYESTIFTVNRRLNAMIKDRMPEELTQEQYATLRYLQGRGQCTSSELAEIFCVGRSSITAIITRLADKE
LIARLPDDKDRRVTYLALTDKGTAAVNEMRLRIQELLAGVIKHFDGQEAITFIETFEKLADVLVQQPDEGSRHKG
>lcl|NZ_CP026520.1_prot_WP_164977163.1_84 [locus_tag=PC41400_RS00495] [protein=glycerol-3-phosphate dehydrogenase/oxidase] [protein_id=WP_164977163.1] [location=complement(120427..122064)] [gbkey=CDS]
MEDRIFDVLVIGGGITGAGIALDAQSRGLKTALVDMQDFAQGTSSRSTKLVHGGLRYLKQLEVKVVAEVGKERAIVYENG
PHVTTPEWMLLPFYQGGTFGAFSTSIGLRVYDFLAGVKKSERRSMLSPQETLDKEPLLKEKHLKGGGYYVEYRTDDARLT
IEVMKKAVEFGVQAVSYAKVEAFEYENGRVTGVRVADQLSGETHLIRAKRIVNAAGPWVDTLRERDGSKKGKTLRLTKGV
HLVFDGERFPLRQAIYFDTPDGRMLFAIPRDGKTYIGTTDTNYKEDIAHPVMTTEDRTYIIRAANEMFPSLRLTEGDVES
SWAGLRPLIHQEGKDPSEISRKDEIFVSDSGLISIAGGKLTGYRKMAETVVDLVVRQLQEEGGIQVSECQTRTLPIAGGD
VGGSARYPQFVEQKVADGQKIGLTVMEARLLAQRYGSNIDKVYRLIDTKRAEAETFRMPDALFGALVYSIEQEMTVKPSD
FFVRRTGAVFFNIKWVRAWKDPVIHYMASRFNWTEEQRAAYTAELERDLHDATVPIDEKTPELVR
>lcl|NZ_CP026520.1_prot_WP_009675896.1_85 [locus_tag=PC41400_RS00500] [protein=glycerol-3-phosphate responsive antiterminator] [protein_id=WP_009675896.1] [location=complement(122278..122835)] [gbkey=CDS]
MSFNNQRVLPAVRKIKDLEKLLTSPFQYIVLLDCHIGMLKSLVDLAKSHGKKPLLHVDLVEGLKNDEYATEYLCQYIKPT
GLISTRASVIMKTKENGLLAIQRLFLLDSSALRKSYTLLERTKPDYIEVLPGVMPHIIKEVGEATGIPIVAGGLIRTVSD
VENALEAGATAVTTSDSSLWKHYQA
>lcl|NZ_CP026520.1_prot_WP_042231320.1_86 [locus_tag=PC41400_RS00505] [protein=MgtC/SapB family protein] [protein_id=WP_042231320.1] [location=complement(122860..123543)] [gbkey=CDS]
MSYIFLQKLFMALILGAFIGIDRQLKHKPLGLKTSMVISVASCLITIVSIESVHKYAIPGTTTMDPMRLAAQIISGVGFI
GAGVILRRSNEIISGLTTAAMVWAASALGIAAGAGFYKEAATAVVLIIMAINLIPLFFKWFGPRKLSERNLAILIIVSPS
ADLSYVIDELHQEFLKVKHIKIKDLKDELRQIQLVVSAPEKHRTTVIYSKIKHITDVVAVEVETISS
>lcl|NZ_CP026520.1_prot_WP_042231322.1_87 [locus_tag=PC41400_RS00510] [protein=glycerophosphodiester phosphodiesterase] [protein_id=WP_042231322.1] [location=complement(123551..124291)] [gbkey=CDS]
MTANKPMIIAHRGAKGMAPENTLGAFKLGLEQGCEGIELDVHLSADGELVVIHDHTLDRTTDRQGAIAEKTLAEIKEADA
GSWYSEAYKGEKVPTLGEVFDLVPDSIMINVEIKAPGNGMEQKLADFLNERGRIDNVVTSSFDHKCLRRLKLIEPKAKIG
LLYANNLLDPTAYARSFDVEVYSLHPHYKAIDAEDVVKAVENGLQVYPYTANEIQDLKELTDKGVSGIITNYPGRLKDLL
QDGAAE
>lcl|NZ_CP026520.1_prot_WP_042231325.1_88 [locus_tag=PC41400_RS00515] [protein=ABC transporter substrate-binding protein] [protein_id=WP_042231325.1] [location=complement(124623..125981)] [gbkey=CDS]
MNKKGMAASLLTAVMLVSAGCGGSDTAGTGDKGGADSGSASGPKKVVFWHAMGGANTKVVDQLVADYNASQDKIKVEAVF
QGSYDDLLSKLKASMGTKEGPSVVQMYEIGSRFMIDSKMVTPMQNFIDADKYDLTQLEPNITGYYTFNDKLFSMPFNTSN
PVLYYNKDLFKAAGLDPEKPPATYEEFQKAAEAISKTGKATGGNFAIYGWFMEQFFANQGAEYINNGNGRKELATQSLVN
SEAGVQTLTWWKNMVDSKAMNNLGRKTDDSKKAFSAGQVGMILDSTAALKGLVDSSKGKFEVGTGFLPKPANAKDGGVVV
GGASLWIMNDRADDEQKATWDFIKFLTSPKEQAYWHINTGYFPITKKAYDEQSVKDNMKNFPQFQTAVDQLHQTKLNTAT
QGAVMGVFPEARQIVEGAIEEVLNNKKSPKDALDAAAKDISSKIEKYNKTVK
>lcl|NZ_CP026520.1_prot_WP_042231327.1_89 [locus_tag=PC41400_RS00520] [protein=carbohydrate ABC transporter permease] [protein_id=WP_042231327.1] [location=complement(126084..126908)] [gbkey=CDS]
MIARRLNNTLLYLLLIVLALVVLYPLLFTVFSSFMTEQEVSKFPPALVPSQLYWGNFKAAIDNVPVFQFILNSFIVSSAI
MVSQVITSAMAAYAFSFLNFTGKNALFMVFLATMMIPWEVTIIPNYLTIKSWGWMDSYPGLIVPFMAAAFGVFLLRQFFL
QLPHELFDAAKIDGCGHWRNFFSIVLPLSKPALSTLAVYEFLTHWNSYLWPLLITNKVSMRTVQIGVAMLQHAEVMSWNM
ILAGITIVLLPSFLLLALGVKQLVRGITAGAVKG
>lcl|NZ_CP026520.1_prot_WP_042231329.1_90 [locus_tag=PC41400_RS00525] [protein=sugar ABC transporter permease] [protein_id=WP_042231329.1] [location=complement(126905..127810)] [gbkey=CDS]
MPSHESVLSKKKRWRENALAYFFLAPSLIIFATFLFYPMLRSVYLSMTLTDPRGRVAEFVWFDNFVELFQSGQFYSDLKV
TLLFILYTVPTSIIWALLLAAFTHNKLKGMKIFQFVFSLPLVISVGTGSIIWLLLFHPTAGMLNYFLSLIHIQAIPWLTD
PAWALISVSLMTVWMNMGFLFIVLSSGMQGVPEEIYESAKIDGSGPVRTFLQIVLPLLSPTLFFALIVSVIGAFQAFGQI
NILTKGGPMDSTNVVVYSIYQEAFVNFRFGTGSAEALILFFIILILTVLQFTVLEKKVHYQ
>lcl|NZ_CP026520.1_prot_WP_042231333.1_91 [locus_tag=PC41400_RS00530] [protein=DUF1129 family protein] [protein_id=WP_042231333.1] [location=complement(128174..128860)] [gbkey=CDS]
MNTSQMSAENAKLQQQLNEENKKYYGGMLVYIRTSDIDERQGEELLLEMLQHLLQAQKEGRSARVVFGTDPLAYCKDLVS
QLPSRGRAERLRMVVMIPWITLTWMFLAYAVTGFLTIMAGAEIGRMNEVSPAFLLIVAVDSILVVKLVLALMNKTAFNGS
ERKFRLFFLLLYIVSIGAVTAAGILLKPYLPVFTVSPWISLLIFAGGYAGQKVMFGGRGKRGGAGAAG
>lcl|NZ_CP026520.1_prot_WP_042231336.1_92 [locus_tag=PC41400_RS00535] [protein=PadR family transcriptional regulator] [protein_id=WP_042231336.1] [location=complement(128857..129177)] [gbkey=CDS]
MLKGILEGCILAIISQNEVYGYELSRKLQESGFDYVSEGSIYPLLLRMQKEKWIEGYMKADSRSGGPPRKYYRLTEAGGE
VLVQFHDNWSAMKRSVDHILEGGQSR
>lcl|NZ_CP026520.1_prot_WP_042231338.1_93 [locus_tag=PC41400_RS00540] [protein=VOC family protein] [protein_id=WP_042231338.1] [location=complement(129338..129748)] [gbkey=CDS]
MAVLRPYLYSENAREQGAFYARALRGEIVSIQTFGDAPAQYGQDRDRVMHLVLKACGLTFYMADSDRVERGSGLDLTLEF
GTDAEAEQVFGALSEGGKVLMPLQRMFWGTMFGRIEDAYGVRWQIATEAAGNQVEA
>lcl|NZ_CP026520.1_prot_WP_156972836.1_94 [locus_tag=PC41400_RS29610] [protein=hypothetical protein] [protein_id=WP_156972836.1] [location=129969..130124] [gbkey=CDS]
MPLTDTGAGEEFYSTAIGATILQPLTVWIPFGMVWWIRRQFGLNEGEEITL
>lcl|NZ_CP026520.1_prot_WP_241688761.1_95 [locus_tag=PC41400_RS30145] [protein=hypothetical protein] [protein_id=WP_241688761.1] [location=130121..130795] [gbkey=CDS]
MSLYADMFISKWHLITLGLILIAWIALIAGGYDKRTILRSGGGAVLLLYIVSTIPIATITHGAHKIDESMRSELDRHYQE
RMSKSEVREDIDRLAVAAGAFSDDDSYDIRVYAGNYSSAYTFRGSLTFTTYDAQGQVVHEKSYDNVIIAPGEKKSSTTTT
RRTHSLHTGIPLPPGRRPGSPDSRVIFWREGLLPASCFTGWRKGKQRQKKNKACPGFKTRACFV
>lcl|NZ_CP026520.1_prot_WP_042231340.1_96 [locus_tag=PC41400_RS00550] [protein=MBL fold metallo-hydrolase] [protein_id=WP_042231340.1] [location=complement(130792..131562)] [gbkey=CDS]
MKIRLIRNATLWLEYGGRCILVDPMLSGAGANPPIPNTANDRRNPLVPLPVPAEDLLKPDLVLITHLHPDHWDAEAAGRL
PKGTPVLCQPGDAEPLAEQGFTDVSPVEEETAWRGLLIARTSGRHGTGDIGAAMGPVSGFVLRAEGEPAVYIAGDTIYCA
EVAEALERHQPAVTVVNAGGAVFKAGDPITMTADDVVAVCRKAPQTRVIAVHMEAINHCLLTREDLRARLEQEGLSERTD
VPADGQWLEESGAASN
>lcl|NZ_CP026520.1_prot_WP_174446830.1_97 [locus_tag=PC41400_RS00555] [protein=Lrp/AsnC family transcriptional regulator] [protein_id=WP_174446830.1] [location=131702..132130] [gbkey=CDS]
MDMFSIDDTDHKILQCLLEDATRSNKEIGLKVHLTGQAVGARIRKLEDWGIIEGRTLKWNPEKLGLTVHAMVTVFLKSQN
AHGDLQTFASSDSRVLEAHRVSGEGCYWLRVQVGDNEELNAFLDDLLKYGNYKVSLSIGKIK
>lcl|NZ_CP026520.1_prot_WP_016423690.1_98 [locus_tag=PC41400_RS00560] [protein=cold-shock protein] [protein_id=WP_016423690.1] [location=complement(132212..132412)] [gbkey=CDS]
MQTGTVKWFNAEKGFGFIEVEGGNDVFVHFSAIQGEGFKSLDEGQRVEFNVVQGNRGPQAENVVKL
>lcl|NZ_CP026520.1_prot_WP_042231347.1_99 [locus_tag=PC41400_RS00565] [protein=GlsB/YeaQ/YmgE family stress response membrane protein] [protein_id=WP_042231347.1] [location=132722..132979] [gbkey=CDS]
MSFLWSLIIGGIIGWLAGMIIGKDIPGGIIGNIIAGFLGAWLGGLLLGNWGPEVGGFYFFPALIGGIVLVFIVSLILRSM
GSKRV
>lcl|NZ_CP026520.1_prot_WP_053228745.1_100 [locus_tag=PC41400_RS00570] [protein=stalk domain-containing protein] [protein_id=WP_053228745.1] [location=133169..134083] [gbkey=CDS]
MKKYVIGFAVGAIVSASTVAYAGNTIQAVLFPAKYQVNGESKTLPDDYKTLNVDGHAYVPIRFAAESLGAVVAYEGKTNT
ISIDNGFNVTTMDSTIRAGHLKVSDTASSTVEGQLYIGQDHWDRMAQARNSIEPGSEVDVKGTILFFDAQGKPLGQAPLE
TKFTAEGDQIKPFKAVADHNVSGYAFASLQAGPVPHPLPVPPNLSISDETGTLGIGTLKLIKQNDFTKVSGWVSVKKPGT
LRFDATLQFYDAEGRDLGTATVKGTSVGSTGESLSATTFETVGKGDLTGYKSVKVKLNSADPVQ
>lcl|NZ_CP026520.1_prot_WP_042231349.1_101 [locus_tag=PC41400_RS00575] [protein=DUF4385 domain-containing protein] [protein_id=WP_042231349.1] [location=134285..134767] [gbkey=CDS]
MAKKFDYSLDYEHLDLRKKPELYTVGRGEQGVLLVEPYKSEILPHWRFKTPEIAKESSEKIYNLFMDYKKSGDFIGMDMA
RKFLQMGYTRARRYTNHKGGRKYSKEDGSILPYQNDPVKAESAAIFKAQWEIAKTDEDYIRMKKEHQDKYEKEDKKQSPD
>lcl|NZ_CP026520.1_prot_WP_042231351.1_102 [locus_tag=PC41400_RS00580] [protein=ATP-dependent DNA helicase] [protein_id=WP_042231351.1] [location=complement(134764..137304)] [gbkey=CDS]
MSGEVRISVRTLVEYAYRSGSIESGFRTATALTEGTKAHQKVQKTYGDSDQKEVFLQTEILLDGMAFVLEGRCDGLLFDD
DGGEVTIDEIKSSTGDPRLVEENRHPVHWAQAVCYAYMYARQHGREHMKVQLTYVQVGTEEQQRYVREYSYNEMEAAVLE
AVRLYAPYAAMQLQNRRERDASIRELPFPYPAYREGQRQFAGSVYKTIKDGHKLFANAPTGTGKTISTLFPSVKAVGEGL
LQRIFYLTAKTLTRTAAEEAFSLMIERGLHMRTVTLTAKDKICFRDEVDCRKEICEFADGYYDRVNGAVLDMLSHEKLMN
RGVIETYARKHKVCPFEFSLDAAYAADAVICDYNYIFDPKVYLKRLPEEVKKQTALLIDEAHNLVDRGREMFSAELTKAP
FLQLKRDFKGRNSAVEAAAKAVNDYFIAFRKEQGSSGSFVLREVAEELPVLLEAFAQQAERELSAPGQGGEAQELLLDTY
FAVQSFLRTAKYYDERYVTLAEVQRSDVRLKLLCLDPSALLAQMGKGYRAHVFFSATLTPFSYYMDMLGGGEDDYTVAIP
WPFRQEQLDVWIEPLSTRYKDRDKTKLPISRLLGKLVREKPGNYLVFFPSYEYMREVAALFLEQERGPQSGAAVDGEAGG
TEAASETAFEAVEVLLQDRLPASDESGGGSILAEEAGSGSSGGSGCSGAETGSGDHSSRMLRILVQQPSMPEEERESFLA
SFRADSSRTLIGFAVMGGIFSEGIDLRGDRLTGVVVVGVGLPQVGPERNIIKEHFDRDGRSGFDYAYVFPGMNKVMQAGG
RLIRTENDYGTLVLIDDRFLQRRYQSLLPPEWRHFTIPGQEKERTH
>lcl|NZ_CP026520.1_prot_WP_042231354.1_103 [gene=lspA] [locus_tag=PC41400_RS00585] [protein=signal peptidase II] [protein_id=WP_042231354.1] [location=complement(137539..138015)] [gbkey=CDS]
MLFYFTAALVAAADQFVKALIRATMAVGESVPFWGSMALTHYENSGMAGSRFQGYARVFAVLAVVFIAIVLYYRRKGEIH
GPLMDASMGFLTGGAAGNAIDRFWFGKVTDFLEFRPGGGILNLADIAINIGVVLFLIGGVLDYYRKKQNPLAAGDEQG
>lcl|NZ_CP026520.1_prot_WP_241688871.1_104 [locus_tag=PC41400_RS00590] [protein=ABC transporter ATP-binding protein] [protein_id=WP_241688871.1] [location=complement(138133..139917)] [gbkey=CDS]
MRGRGPVVKPKNFKGTLRRLWAYFGSERKLLTVIFALILIGAALTLVGPYLIGVSVDAMSHPEQGVDFGLLKTMVLVLLL
AYLIDAVFTFLQSWLMAGVSQRIVMRLRGALFAKLQKLPVAYFDTRPHGEMMSRLSNDIDNVSSTISQSTTQLMSGVIAI
VGSLAMMLVLSPTLTLASLVTVPILFVLTRTITKKTSVLFKNQQIQLGKLNGHIEETISGLQVVKAFNHEEKAIAEFERV
NAELCEVGLKAQIRSGFLMPLLNVINNIGFAAVALVGGILAVRGDITVGVIASFLSYSRQFVRPLNDLANIFNILQSGVA
GAERVFEVIDEEEEPFDSEGAVELLHPKGHVVFEQVSFGYRPDVPILNNVSFDSAQGSSTALVGPTGAGKTTIVNLLTRF
YDVTGGRILIDGRDIREYTRDSLRRTFGIVLQDTYLFSGTIKENIKYGRPDATDAEVEAAAAMANADVFINRLPAKYETL
LSENGGNLSQGQRQLLAISRVILAKPSILILDEATSSIDTRTELHIQDALLKVMQNRTSFIIAHRLNTIRDADTIMVIDH
GEIVEKGAHDSLIGQQGVYYRMFYNQFKNIEGAV
>lcl|NZ_CP026520.1_prot_WP_042231359.1_105 [locus_tag=PC41400_RS00595] [protein=ABC transporter ATP-binding protein] [protein_id=WP_042231359.1] [location=complement(140009..141793)] [gbkey=CDS]
MPISRKYISKYWKPFSAAVAFLTLEALCDLTLPTLMSKIIDVGVGGKQMDYVWRMGGIMLLVTALGAISASVRNVISSHV
SQKFGAELRSDLFRKIQSLSFENIDRFDRASLVTRLTNDVTQVQMFVNGLMRVFVKAPLLCIGSLIMAARLNPPLSVILA
VVVPVVGILIAINMKVGFPYFIKVQQALDRVNGIMREYLSGVRVVKAFNRFDYEVGKFGRANEEYRSRSVTAMRVMSAFN
PGITLTVNFGIVAVLWLGGLRVDQGQMQAGHIIAFVNYMTQILFSLMLISMVFNMFVRARASAGRIGEIFAEENTMAAQI
PVTTAAGGSSASSADKPASGTLTGMQTGGSVEFENVTFSYEGASGEPVLRRITFSCEPGQTVGIIGSTGSGKSTLVSLIP
RFYDAVSGTVKVNGTDVRELEPSRLREKIAVVPQKTVLFTGTVADNLRWGKEDASPEEMEHAARMAEAHEFIAASPEGYE
TRLGQGGVNFSGGQKQRLSIARAIVRKPDILILDDCTSAVDAATEARIKDALKKYAEGLTCLLIAQRITSVMDADKIVVL
DRGEIVGSGTHDELLGSCRVYREIFQSQMGKEVG
>lcl|NZ_CP026520.1_prot_WP_042231360.1_106 [locus_tag=PC41400_RS00600] [protein=NAD(P)-dependent oxidoreductase] [protein_id=WP_042231360.1] [location=complement(141964..142599)] [gbkey=CDS]
MKIAIIGASGKAGSRILKEAQERGHEVTAIVRDAAKVSAEGAALLEKDVFDLTASDLKAFDIVVNAFGAAPGQEHLHVDA
GNVLIEALQGAPETKLIVVGGAGSLFVDEAKTTRVLETPDFPAAYFATASNQGKNLEILQNTAGIKWTFVSPSAIFALGK
RTGQYRTGKDNLLVNSKGQSYVSYEDYAAAVLDEIEQPKHENERFTVVSES
>lcl|NZ_CP026520.1_prot_WP_255246850.1_107 [locus_tag=PC41400_RS30510] [protein=hypothetical protein] [protein_id=WP_255246850.1] [location=complement(143028..143159)] [gbkey=CDS]
MVALVAFYDVLRWTVTLLSLVLGIRKLYRWARRKFRNKRKNHR
>lcl|NZ_CP026520.1_prot_WP_042231361.1_108 [locus_tag=PC41400_RS00605] [protein=DUF3298 domain-containing protein] [protein_id=WP_042231361.1] [location=143475..144605] [gbkey=CDS]
MKKSFKLLTLGLAGSCLLAASAGTSYANNAVQVPAATSVAAEPVSAQASVKNGAVQVTTKEIKEDSPELSVDLQIPVIQG
MKDTQYQDELNDIIYRHAMKDVEAVRKQAQRDLAAANGSASELAMPYSVDLIAEVSASGGQEDANRLSLRVATSVYTGGA
HGMPRVDTYNVLNAEEAQKIELKDLFGDNYKAIIDKQISQEIAKHPDDFFEDAFSGITDTQSFQIRDGKAVIVFSPYDIA
PYAAGIQEFSIPIAPGGNSSAPSSLVVGGEVLSAKQGSLYTTAEGTNMVPLRVAAEKLGFQLKWNQEQWAAELSKGAVWT
SVYKGKDAYTINKMAPVSLGEAAVIKEDGNMYVPASFFSKILKADVTVTGDQLVIK
>lcl|NZ_CP026520.1_prot_WP_042231362.1_109 [locus_tag=PC41400_RS00610] [protein=NAD(P)/FAD-dependent oxidoreductase] [protein_id=WP_042231362.1] [location=complement(144810..145991)] [gbkey=CDS]
MSKHILILGGGYGGLLSALTARQHLTAEEATITVVNRFPTHQIITELHRLAVGNLHEKAVALPLEKLLRGKDINLEIGTV
ETISADSKRVILSNGKSFDYDKLVVALGSETAFFGIPGLQEHSFTLKSVNDANRIREHVEARIEAYSQTKNKADATFVVG
GGGLTGIELVGELADMLPGLCRSKGVDFNDVSLYCVEAAPSILMGFHPELIERATTSLQKRGVTFLTGVAITEMQESVVS
LKDGSTIETSTLVWTGGVQGTPVVANSGIEVNRGRATVNEFMQSTSHPDVFLAGDCAVVFPAGAERPYPPTAQLAWQMGE
VVGYNLFAAQKGAPMDVFTPVFSGTLGSLGRKDGIGSIGGNQTRLKGLPATLMKEASNIRYLSHIHGLFALAY
>lcl|NZ_CP026520.1_prot_WP_042231363.1_110 [locus_tag=PC41400_RS00615] [protein=DUF1641 domain-containing protein] [protein_id=WP_042231363.1] [location=complement(146029..146454)] [gbkey=CDS]
MSETSNQTPSTTEKRETQDVLDQLMKPEVQESLTVLVENLPKLAEMVTILTKAYDLAQSVASDQVLIDDLKGGMNEFVKP
ITEKAKGVAAAAIEASDRAQAEHSTIGLFGLLKMLKDPQVQKTFRFVQAFLDVQAERQQQR
>lcl|NZ_CP026520.1_prot_WP_042231364.1_111 [locus_tag=PC41400_RS00620] [protein=threonine/serine exporter family protein] [protein_id=WP_042231364.1] [location=complement(146979..147404)] [gbkey=CDS]
MIAQLVTSYIASAAFGIIFNVPRNTLIQCGFVGMMGWLVYVALVRSDTNAIVATLAAAFFVTVISRVFARLYKTPIIVFS
VAGIIPLVPGGMAYDAMRSFVENDYNLAVQLAAKAFMISGSIAAGLIFSEVMNQLFKRPKV
>lcl|NZ_CP026520.1_prot_WP_042231365.1_112 [locus_tag=PC41400_RS00625] [protein=threonine/serine exporter family protein] [protein_id=WP_042231365.1] [location=complement(147419..148180)] [gbkey=CDS]
MEQASGPGQTYKIVDVCLLAGKIMLQNGGETYRVEDTMTRIAASYGLHHTHSFVTPTGIIFSVDGSDPTKLVRISTRSTD
LHKVTLVNGLSRKISNGELDLQEAHRLLLKVDRANLAYPVAIQIAAAAVSSGCFTIMFNGGWPDFLPAVVTGGLGFIGFL
IFHRYVLIKFVAEFLASFIIGMLASLFIWTGLGQELDKIIIGSVMPLVPGLLITNAVRDLMAGHLVSGLSKGAEAFLTAF
AIGAGIAVVFSFY
>lcl|NZ_CP026520.1_prot_WP_042231366.1_113 [gene=rluF] [locus_tag=PC41400_RS00630] [protein=23S rRNA pseudouridine(2604) synthase RluF] [protein_id=WP_042231366.1] [location=complement(148236..148925)] [gbkey=CDS]
MRINKFISETGICSRRKADELVQAGRVTINGAPAELGSTAEPGDDVRIDGKPLGTKKKSVYIVLNKPVGITCTTERHISG
NIVDFVRHPERIFPIGRLDKDSEGLILMTNDGDIVNKILRAENNHEKEYIVTVNQPITNSFVQGMSTGVRILGTKTKPCK
VTRINDHTFRIILTQGLNRQIRRMCQAFGYQVHRLQRVRIMNIKLDGLPKGKWRDLSESEFTRLMELLD
>lcl|NZ_CP026520.1_prot_WP_241688762.1_114 [locus_tag=PC41400_RS00635] [protein=hypothetical protein] [protein_id=WP_241688762.1] [location=complement(149161..149460)] [gbkey=CDS]
MESKKKFGTGSTLLSSLGVLLLVGSYLISENPAGSESAGIRLLFYTAIGSLAASLVLGYAGMLKEERGFLKFAVTGSIFL
IAGALIAALLLLFAGFQEP
>lcl|NZ_CP026520.1_prot_WP_042231368.1_115 [locus_tag=PC41400_RS00640] [protein=YegS/Rv2252/BmrU family lipid kinase] [protein_id=WP_042231368.1] [location=complement(149613..150509)] [gbkey=CDS]
MFGKAILIHNGSAGQAETGSRLGAVIGPLEAGIEELTLVQTKEPGDAERICRERGEETDLLLVLGGDGTVHECVNGLAPL
ADPPVVGILPGGTCNDFSRALYIPQELGRAAEALMQGRTRPVDIGRANDRYFSNFFGIGLITDASVNINPQLKGTIGKLS
YFISTLQTAASADPFRFTLEHDSGRTDGEAVMLFVANGRFLGTRPLPLAADALYDGVLDVAIIREAGLPLLKELFKIKNF
GEWDSEKSSFEFIRTTSLKLTTDQPMQADMDGEIYAHTPASISVLGKKLTFLVGEEQP
>lcl|NZ_CP026520.1_prot_WP_042231369.1_116 [locus_tag=PC41400_RS00645] [protein=ABC transporter permease] [protein_id=WP_042231369.1] [location=complement(150854..151594)] [gbkey=CDS]
MAAFFKILGAEWLKLRKTNIWLLIFISPLLAFATGLALPVPANEPLIAWTLLLSTMALIHAMLFLPLLTGVFAAFVCRYE
HTGGGWKQVLALPVSRSHLFMAKLLLVTGLLAVTQLLFLGGLVGVGCIKGLGASIPWSQLLWPVLGGWLATLPLAALQLW
VSVAWVSFAAPLALNVIFTLPNILVANSETYGPFYPWAQPFLAMLPASDTSLGGLHVSFETLIYVILGGFVVFFLSGLSY
FQRKEI
>lcl|NZ_CP026520.1_prot_WP_042231370.1_117 [locus_tag=PC41400_RS00650] [protein=ABC transporter permease] [protein_id=WP_042231370.1] [location=complement(151596..152309)] [gbkey=CDS]
MIGRVLAADFLKIRRTLAWFLVLLGPVGVVGLQAVNFGLRYDYLTKQYANDLWYGLLSNIQFLAAPALLLGMTLLTSMIA
GYEHQRNSWKQLLALPVSRFSVFTSKLLLVLILLLVSCLLLAAGSVLLGFLLGFGMNVPVMHVLRISLFPLLAGIPILSI
QLWLSVAVRNQAIPLTAGILGTVLSLFSIRLPDWFLWKWPLLVNEAGRPELNVWLGIATGIVVFLIGLTHFTRQDVN
>lcl|NZ_CP026520.1_prot_WP_042231371.1_118 [locus_tag=PC41400_RS00655] [protein=ABC transporter ATP-binding protein] [protein_id=WP_042231371.1] [location=complement(152306..153220)] [gbkey=CDS]
MNEMIIQTFGLSKQYKLRSAVDNVNLQIRKGDIYGFLGPNGAGKTTTIRMLLGLIRPTKGTVQLFGKDLRKEKLAILRKV
GSLVEYPSYYGHLTAYENLEAIRRILDVPKQRIDEVLAIVRLTKEAKRPVKGYSLGMKQRLGIAAALLGSPELLVLDEPT
NGLDPSGILEIRELIKSMPQQYGVTVLVSSHLLSEVDQMASRVGIISGGSMIFEDTIESLRKRAGGQIRIAVDAPQQALR
TVASAGCFAVADESSLLLEEAPDEKIAQVVAGLVQAGHSVYRVEEKKKSLEDFFLDLVGEGESL
>lcl|NZ_CP026520.1_prot_WP_042231372.1_119 [locus_tag=PC41400_RS00660] [protein=HAMP domain-containing sensor histidine kinase] [protein_id=WP_042231372.1] [location=complement(153298..154659)] [gbkey=CDS]
MKWRHSLLVKYLCIVVFAMLIWPIVLPMAAVLYYLPHSFKTQEENRKTNPYARGNDIEKSWHGEAKKLQGAGDEGIDKRI
RELKEKYPKADLFWVDGSGKTRLTLPERSDLPAVWEAADTIAFMKKSYNGDPFTTVAFVGEDASQGFMVMQIPRVLMKEE
RPLYSDRYVIGIMLLAFGLFLFMSWLFFYRIRKRLVHLEEAMGLPDGREIPHPIAVSKKDEIGQLELAFNRMIEELTTSR
QRERKEEGLRKQLIANLSHDLRTPLTIIRSHAYSLQKEPISASGKSSLALIETKSDDLNRLIDNLLSYTLLSAGKYPLQR
TEIDVARLARASAAAWYPVFEEEGMVMDVGVTEHPVYWKADPQWMNRILDNLFQNVLRHAKSGGYIGLRTEEREGGTALM
IEDKGPGMKAPSAEKGAGIGLTIVALMVKEMHLDWDVVSSREGTAIYLFPRVR
>lcl|NZ_CP026520.1_prot_WP_042231373.1_120 [locus_tag=PC41400_RS00665] [protein=response regulator transcription factor] [protein_id=WP_042231373.1] [location=complement(154656..155333)] [gbkey=CDS]
MAVKQILYIEDDPEIGKWVNSHLIERGYEVNWLKSGEKAAEFWEKSDLVILDVMLPGLDGFSIGQRMKKNYPDTPVLMLS
ARTSIDDKLQGLQFADDYVTKPFHPDELTARVEVLLRRTGKLAQAELTLHHLQVNMEENRIVDTRTGEEIILTGKQFQIF
MYFLRHPNQILTKEQLYEGVWGDPYMDGDKTLMVHIRYLREKIERDPSQPEIVETIRGIGYRVRA
>lcl|NZ_CP026520.1_prot_WP_042231374.1_121 [locus_tag=PC41400_RS00670] [protein=protein kinase] [protein_id=WP_042231374.1] [location=155560..156471] [gbkey=CDS]
MRMPKRFKHLYTYLHKCFGKSLRSMRGLPFRKGAVIAGTYKIMKKLGAGSFGVAYLALKTDNGQQVVLKRVAPVRGGKKR
AEYIFGQETAMLRQLRHPDIPAFYKAFRWRGEFCFTMEYVRGSSLDVLLFREERAFSEKEALLLIRRLLELAGMLHGQGI
VHRDISIANVLLDGDRLKLIDFGLARALSSGGAASDSPAGEHLPRWDADVDDDDPSEKKFRRAVHVTGDFYAAGHLLLFL
LYSTYSAEDESLAEGAGLSWEKELLLHPGTKKLLRRLLLTEQPYEHASEIQSEIDGILAELEG
>lcl|NZ_CP026520.1_prot_WP_042231375.1_122 [locus_tag=PC41400_RS00675] [protein=hypothetical protein] [protein_id=WP_042231375.1] [location=complement(156475..156654)] [gbkey=CDS]
MLKKMLTKLLGSDKNKGSHYRPHSSSDDFRRHSRKHYNGHNHYGHNHYKKKRSSSSYSS
>lcl|NZ_CP026520.1_prot_WP_042231377.1_123 [locus_tag=PC41400_RS00680] [protein=TPM domain-containing protein] [protein_id=WP_042231377.1] [location=complement(156869..159283)] [gbkey=CDS]
MKRFRVIGLALLLIFLTCTTVFAAAVPRNDGIVTDPVGLLTQTQARQVESGLKNKDYEVFVLTAKGLNEREGERLANEAY
DQWELNRNQLMLVITTQPNFVHLVYENEQLAAAVSRSKAKDAKGIVDLKFVPAAGEGKIGEGILAVSGYVNSLAGAGASS
GNAGDSGANDARGGGMASIFPVVLGLAVIGAGIWAYWQYKRRSAARKRLTEAKQLLDEMQPVLNGMLFSEVFQDLEKQFL
QGETKEKAAELENAVVRLQGESEELGGRLAELKIPFLINAKAEAETRKLFQAVQEFAGRIGPYGTQLAEIEKQSTEVRKS
VDRAKNGIAEAEEAVEALARRTGYPLTVLRKHVEQAKVAYAKADELDEFDFMQAATSVEDALRELDFIRMSVKDFEALAE
RQPGFLPEIQALEQELRERARQEGLLLTDGDPYALLKQAREETARLGTLLEAGDTEEGGRSAAAIEEGMEAARSLVEAMI
ADRESARKTAHEVEELTGDLGSFDRAYPEESLKIARDYAEVHRREQQADYTRMTRARDELDRRLTEIRSGLAPDVQEYRS
AREAGTEAQALMTEIRELHTRILGYHDSLDARVRTVRHALDEERGRLRQAASAFGELQVESPELSAMIRDGESRLAELEQ
LAQAPVLDVDLLEERRRGLAVQAGQAAERVQQLLREKEETLRTIRQLEGEYRSRHARYGGSMSLSPYSSGYDTLMEESRR
LIARGLFAEAMQRISSGQQLLEQMDRDYQRRMYEEQLRRRGPGGPGGGGSSGSSGWGGSGGRSSGSSSWGGGSSGGRSSG
SSKW
>lcl|NZ_CP026520.1_prot_WP_042231379.1_124 [locus_tag=PC41400_RS00685] [protein=LysR family transcriptional regulator] [protein_id=WP_042231379.1] [location=complement(159387..160259)] [gbkey=CDS]
MTITQFQVFVKIAETGSFTKAGLELNMTQPAVSHAISSLEADLGIHLMIRDRKNGIALTHIGKRVLIQMREILQRVENIE
QEIAAEKGLEVGTVRIGAFPTAAAHFLPKIMSTFKQKYPNLQLVLHEGTIAEVKEWLSTRFIDVGLVIFPDADMEIVPLF
RDKMVAVLPETHPLGGRDFLTISDLSGEPMILCRGGYEAPIIDLFQKSGAELQLAFAASNVATVLNMIREGLGLAILSRL
SLSALPPGLVTLDLEPRVCRDISVAVPSLKEASLAVHLFIQMARELFPRE
>lcl|NZ_CP026520.1_prot_WP_042231380.1_125 [locus_tag=PC41400_RS00690] [protein=DMT family transporter] [protein_id=WP_042231380.1] [location=160357..161307] [gbkey=CDS]
MERSSRKRTAFLLAFLVLMWGINWPLTKHALIYTPPLLFAGIRTLIGGFLLLLVAIPRYKKLNFKQTWYIYLISAVLNII
LFYGLQTVGLGYLPSGLFSAIVFFQPVLLGILSWLWLGESMYGLKMFGLLLGFAGVATITSAGLNGHISFTGILLALGSA
VSWAFGTVFMKKTGDRVDSIWMVTLQLLMGGVVLTGTGLSTESVSDIAWTGPFIFSLLFISVFVIALGWLAFFTLVGSGE
ASKVASFTFLIPLISITASAIFMNEAITINLLAGLMLIVVSIVLVNVPLRSFRAAKAKKSKDIPPCASLAQARDYK
>lcl|NZ_CP026520.1_prot_WP_053228747.1_126 [locus_tag=PC41400_RS00695] [protein=glycosyl hydrolase family 18 protein] [protein_id=WP_053228747.1] [location=complement(161406..166415)] [gbkey=CDS]
MKALKQSFVWRWLVLLLCTSMVFSTYLTVVSGQASAQPADIKGHWAEKELGEWHDKGLIQGYSDGSLQPDNAITRAELMA
LINRYYKLTASASIQFTDVTAEQWQYEQISKAVAAGYVDGYSDGSIRPDQPVSRQESAVMLSKLLKLDEASEDALKNIKD
AASIADWSKGAVSAMVAKGVIEGYEDGTFRPAGKLTRAETVVLLGRAPSLQTVSYDKAGTFGPASGTETVNGDVAVNVSG
VTLQNTVIQGNLLLGKGIGNGDVTLKNVKVTGTVTVNGGGENSIHLVDTVLIKVVVDKETGVVRLVAQGNTSVQETYVQS
SAKLEESSLTGAGFIDVIFAKVLSAGSSVQLIGSFNSLHVLASGIKVEISKGSIGKVIVDGSDTTLAVNNDASIVSLVLN
DLAKITGYGSVENATVNEGGKGSSFEKAPVKVDGPQKGSIIIGLPIGGGWGGSGGSSGGGSTETAKKVIAYVPGWKNWSA
SNPIDAAKLTHINYAFTHVKDNKIVSLPAQNDDANYAYLQSLKKDNPKLKILPSVGGWGADGFSDAALTESARNTFADSI
IEYIKKYKLDGVDLDWEYPTQSADGVMKARPEDKQNFTLMLQTIREKLNALGSQDGKYYELTIAVGATKKYLDGVEIEKI
TPLLDNINLMTYDFHGGWDTKTAHHTNLYAGDLSVDSSVKLYLANKVPASKIVIGGAFYSHLWTDVEAAGGKIVGQKAVG
NGATPTYNEIISNYNEASGYKRYWDSEANAPYLYSEQTKVFLSYDDPQSLHEKGSYVLNQKLGGAMFWEYSQDQTGALLD
ALVKGVQGKTLNPDTTTVPAAPAVTGVSEGQTYKTGVLPNWKDAAGTASGATLNGAAYTKGTGITEAGEYTLVVTAAHGK
SGKTASTTVKFKVEPQTKVIAYVGGWKNWSEAKPVDASKLTHINYAFTHVKDNKIVPIEVQNDDANYAYLHSLKAQNADL
KTLISVGGWGADGFSDAALTEASRQTFTDSIIDYVKKYNLDGVDLDWEYPTQSADGVMKARPEDKQNFTSLLQLMRDKLN
ALGLETNRYYELTIAVGAGSGYLKGVEIDKITPLLDNMNLMTYDLSGGWVPKTEHHTNLYGSGNSVDQSVQLYKNSGVPS
SKIVIGAAFYSHLWTNVESSANNGLGQKAVGTGATPTYSVTLATYSEANGYTRYWDQNAKAPYLYNAQTGIFISFDDPES
MAEKAAYTLKEKLGGAMFWEYSQDETGALLSALVDGLKGNPYTPDTTSVPAKPQIANITEGETYTTGVLPNWTDAPGTVG
VATLNGEEYIKGTGITEAGDYTLVVTAVHGKSGKTASTTVKFKVDPRKKVIAYIAGWEDWSVTNSVYANKLTHLNYSFTL
VKDNKIITRPEQKDDQNYQYIHSLKAQNPNLKVLFAVGGWGADGFSDAVLTNEARDTFSNSIIDYIKKYKLDGVDIDWEY
PTISADGLMKARPEDKHNYTLFLQMLRDKLNQLGLADNKYYELSMAAGAGPSHLQALEAEEISKYLDNFNIMNYDYSGGW
VQKTEHHTNVYGPGLSMETVVKRFKDAKVPANKIVVGIAFYSHIWTDVQSTANNGLGQAAKGSGSTPTYNEILEKYNAAN
GYTRYWDDAAKAPYLFDGSTWLSYDDPESIKAKAQYVQDQGLGGAMFWEYSMDKSGALLDALIQVIPVK
>lcl|NZ_CP026520.1_prot_WP_241688763.1_127 [locus_tag=PC41400_RS00700] [protein=bifunctional 2',3'-cyclic-nucleotide 2'-phosphodiesterase/3'-nucleotidase] [protein_id=WP_241688763.1] [location=complement(166956..171029)] [gbkey=CDS]
MKQLATASLAFGIISSQSLGGIVQAQTSASPTVNLRIMETTDLHTNLVDYDYYKDAAAASVGLAKTATLIKEARKENPNT
FLVDNGDLIQGTPLGTYMAQINQLKDPGAIHPVYKAMNLLDYDAATFGNHEFNYGLDFLDKSIKGAKFPYINANVYIDDK
DNNPDNDVNKFTPYLIMDKTFKDENGNDQKIKIGVIGLVTPQIMEWDKVNLAGKVVAKDIVATAKKFVPEMRTKGADIVI
AMAHSGFDGTAAANDQLAENVVLPLSKVPGIDAITFSHSHKVFPTPDVKKLDALFKDANGNPVAGVDNAKGTINGVAAVQ
SGYGGSNLGLIDLKLVQEGGKWKVSNSQSSTRYIYDSTAKKALVETDPAIVQAVAAEHEGTKKYANQPIGETTAPIHSYF
ALVQDDPSVQIVTNAQKEYVSKYISTTLPQYKDLPILSVGAPFKAGRNGPSEYTDIAKGPIAIKSASDLYLYDNTLKAVK
VKGSGVKEWLEMSAGMFNQIDPAKTDEQQLLNSSFQVFNFDVIDGVTYQVDVTKPAKYKVDGAVNDASSSRIVNLQFNGK
PIDPNQEFIVVTNNYRASGGGNFPGLKNAELIVDSADENRQALMDYISEQKTINPAADKNWSIAPVAGNVKVTFTSSPSA
VTYLPENPHIKSTGVTTDKGFSVYSIDLSGKGQPQPDNKNVKVQLLGINDFHGQLDTTTETASGKVGSAPYLATYLKERK
KTNPNTLLVHAGDSVGATAPVSSMERDKPTLEFMNMLKFDVGTLGNHEFDQGVGAMLAQINGGTDTKNKDVTFDKLNFPY
VNANVVDEKTGENILPSYVIKEVGGQKIGFIGVATNSTPSKVSPDGLKGVKFVEQAPVVNKAVQELKAKGVKAIVVLAHD
PASVKDNVASGEAVDLANAIDDEVDVIFAGDNHAKVNTVVNNKLIVQAYSYGTAFADVDLEIDPATGDIVKKTAEIVDVK
QEGVTPDADTVALINKYLDKYPVLKQPVGTTDSVISRNNAYNEEAPLGNLIADAMRDGMNTDFAFMNPGGIRADLPKGDI
KFSDLAKIQPFGNTLVKMTLTGSQIKTLLQQQWGKKSDGSADTKTLQISGLKYTADFSKPAAERIVKLTKADGTPIEDAA
EYTVTVNNFMAAGGDNYKVLVEGKNSQAGLIDLDVFYNYITKTFKGGKITASLDGRIVNLKSFADVPQTHWAYKVIQELS
GKQIVDGVADNRFEPEQDVTRAQFVNMIVKQMKLVSSKPSAFKDVPAGSWYAEAVAAASEKGIVEGSSADTFAPDQKITR
EEMAVMITRMYAQLQGTTGTPKAIFTDGGEISTWAKTAVETAAEKGLMTGRADGVFAPRAKATRAEAAQVIYNGITK
>lcl|NZ_CP026520.1_prot_WP_042231384.1_128 [locus_tag=PC41400_RS00705] [protein=alpha-amylase] [protein_id=WP_042231384.1] [location=171321..172781] [gbkey=CDS]
MTRNHTMMQFFEWHVPADGSHWKRLKELAPELKKKGFDAVWIPPVTKGQSDQDTGYGVYDLYDLGEFDQKGTVRTKYGTK
EELLEAIKACQDNEICVYADLVMNHKAGADEKELFKVVQVNDQNRNEVISEPFDIEGWTKFTFPGRGDKYSAFKWDFGHF
NGTDYDAKEDKTGIFRILGENKSWNDNVDDEFGNYDYLMFANIDYSHPGVRKEMINWGKWLVDTLQCDGYRLDAIKHINH
DFIREFASEMIKERGENFYIVGEFWNPEVAACREFLDNVDYKIDLFDVALHYKLHAASQAGKDFDLTTVFEDTLVQTHPL
HAVTFVDNHDSQPNESLESWIEDWFKQSAYALILLRKDGYPCVFYGDYFGIGGEAPAEGKAAAIDPLLYARQHKAYGEQE
DYFDHPNTIGWVRRGDAEIERSGCAVVISNGDDGEKRMFVGEERAGQVWLDMTGTREDEITIGEDGFAGFPVNGGSVSVW
AQPDES
>lcl|NZ_CP026520.1_prot_WP_042231386.1_129 [locus_tag=PC41400_RS00710] [protein=metalloregulator ArsR/SmtB family transcription factor] [protein_id=WP_042231386.1] [location=complement(173332..173724)] [gbkey=CDS]
MGAAEPKQDVFQAIADPTRRKVLNLLAGQEMPVNAISGHFPMSRTAVSKHLRILADAGLVQERKVGREKRYRLQPEPLLE
LKRWLAYYERFWENRLSVLKHLVEAEEPGESGSGSSPALGLKAAEPGGRP
>lcl|NZ_CP026520.1_prot_WP_042231387.1_130 [locus_tag=PC41400_RS00715] [protein=SRPBCC domain-containing protein] [protein_id=WP_042231387.1] [location=complement(173726..174148)] [gbkey=CDS]
MTNTPNTLPDIRQTQLFNAPIQKVWNAVATSEGIAAWFMPNDFQPELGYEFHLEAGPFGKSPCKVTELDPPNRLSFRWGK
DWTLTFELADLDGRTEFTLIHAGWDPEKVTEFGQSHAEVRERMSGGWVGLVEALRKQVEA
>lcl|NZ_CP026520.1_prot_WP_042231388.1_131 [locus_tag=PC41400_RS00720] [protein=hypothetical protein] [protein_id=WP_042231388.1] [location=complement(174175..174558)] [gbkey=CDS]
MSEELLKKLRYKEGPALVLNAPEGYRLGIETEAANQEDKAAFVQLFVNHGQEVEEWVPKVIPMLSEDAVFWITYPKQSSK
VKTDINRDTLAARVQNNTEYRPVSNVAVDDKWSALRFRRSDLVKPKK
>lcl|NZ_CP026520.1_prot_WP_042231389.1_132 [locus_tag=PC41400_RS00725] [protein=GyrI-like domain-containing protein] [protein_id=WP_042231389.1] [location=complement(174677..175168)] [gbkey=CDS]
MNMKSVFANKPSFRLAGVSVRTTNAAEAGPEGKLSQLWQQFFGSDIASRPGIENPHLIYGLYTDYESDASGEYTALLGHE
ISEETTDGASSGLQFADVPAARYMIFETRKGPVKDVVPQLWHEIWAYFQNAAEQRTYTGDFEVYDGRGFDPENCVVQVYI
AIR
>lcl|NZ_CP026520.1_prot_WP_042231390.1_133 [locus_tag=PC41400_RS00730] [protein=YafY family protein] [protein_id=WP_042231390.1] [location=complement(175235..176173)] [gbkey=CDS]
MKIDRLLAMTVLMLNRGRISAKELAERFEVTTKTIYRDMETLSRAGIPVVAHQGVSGGFEIMEPYTLARQLVSLKEISSL
LAAVKGVSTALDDQTYANLAEKVQGLLLRTDTADWERRKDEIVFDFHPWGQGAAAKNKVSLLKRAIEESRRAAIVYVNTN
GTETERIVEPAALILKGSVWYLHAYCTMRKEFRVFRLSRIADLSVLSQASEPREVPPLEGYGWNRQWLAHKEVEMTLHFR
PEVRHRVGDAFPPGLVSLLPDGNLCVQGVFALDEWFYGMLLSFGDSLKIEHPPSLADEILERARKILLRYEN
>lcl|NZ_CP026520.1_prot_WP_042231391.1_134 [gene=hxlB] [locus_tag=PC41400_RS00735] [protein=6-phospho-3-hexuloisomerase] [protein_id=WP_042231391.1] [location=complement(176464..177021)] [gbkey=CDS]
MQTTQYLAEIREELHRTAEAVVSREAEELADAILNAGKIFAAGAGRSGLMMKAFAMRMMHLGFDAYVVGETVTPNLIKDD
VFIVGSGSGETKSLVSMAEKAKNLGASVAAITIFPESALGRLSDIAVKLPGSPKDKSDNPYPTIQPMGSLFEQTLLLFCD
AIILRLMEKKGYTSETMFGRHANLE
>lcl|NZ_CP026520.1_prot_WP_042231392.1_135 [gene=hxlA] [locus_tag=PC41400_RS00740] [protein=3-hexulose-6-phosphate synthase] [protein_id=WP_042231392.1] [location=complement(177028..177660)] [gbkey=CDS]
MELQLALDLVNIPEGIELVKELEPYIDIVEIGTPVIINEGLKAVKEMKRAFPNLKVLADLKIMDAAGYEVMKASEAGADI
ITILGAAEDMSIKGAVEEARKQNKKVLVDMICVKNLEERAKEVDAFGVDYICVHTGYDLQAVGKNSFEDLQTIKGAVKRA
KTAIAGGIKLSTLPEVIKAQPDLVIVGGGITGQDDRKAAAAEMQKLIKQG
>lcl|NZ_CP026520.1_prot_WP_042231393.1_136 [locus_tag=PC41400_RS00745] [protein=helix-turn-helix domain-containing protein] [protein_id=WP_042231393.1] [location=complement(177962..178324)] [gbkey=CDS]
MGDQLFKQSVELTLKVVGGKWKPVILCHLTEGTHRFGELKREMPDITQKMLTQQLRELEQDEIIDRKVYNQVPPKVEYSL
TPYGLTIKQVLNVMSEWGANHQKKMQAAPSEPPHTEAAEP
>lcl|NZ_CP026520.1_prot_WP_042231395.1_137 [locus_tag=PC41400_RS00750] [protein=LLM class flavin-dependent oxidoreductase] [protein_id=WP_042231395.1] [location=178631..179662] [gbkey=CDS]
MADTNQPAASITRIPLSILDLAPVVLGATPADSLKNTLDLAQHAEQWGYHRYWLAEHHNMPGIASSATSIVIGHVAGGTK
KIRVGSGGIMLPNHAPLIIAEQFGTLESLYPGRIDMGLGRAPGSDQLTARALRRGPESDGQHFPQQLEELRGYLNPGPGA
SPGRVRAVPGEGLNIPIWLLGSSGFSAQLAGQLGLPFAFASHFAPDYLVAALELYRSSFRPSEALKEPYVMIGVNVFAAD
TEEEARRLATSHEQQFLKLIRGQTGLLNPPVDNMDELWSPHEKAIVRKQLSITAAGTKETVRERLDYLLSETQADELIVT
SQIYDHGARLRSYELLAEAAGER
>lcl|NZ_CP026520.1_prot_WP_042231397.1_138 [locus_tag=PC41400_RS00755] [protein=cytochrome P450] [protein_id=WP_042231397.1] [location=179949..181154] [gbkey=CDS]
MAVKEQLFINEITGFKMKDEFWNPYPWYKKMREESPVFYDERQDTWNVFLYDHAKQVLNDYKLFPSKRDRSLIPVPLSDN
KVNLIFSDPPDHRKRRGLLSKAFTPHSLEDWKPRIQNIVNELVANIAKKKSVDLVQELAVPLPVTVIADLLGVPSKDWQQ
IKNWSDILFMPHDFQHIEELQAAKARASKEFVEYLLPIVAQKRAQLQDDIISDMILAEYEGAFFTDAEVAQTAMGLLGAG
NETTTTLLTNMFYSLVKDQPGAYGELRKDRSLVPQAVEETLRYRFAAALDRRIGEDTDVFGPKMKKDQLIIAWVSSANRD
ENRFTDGESFDVRRKDIKHLTFGSGHHFCLGAPLARMEAVQALETFVERFSDYSIPDDFRPQDHLPPNNHNLNSLPVVVE
E
>lcl|NZ_CP026520.1_prot_WP_042231399.1_139 [locus_tag=PC41400_RS00760] [protein=alpha/beta hydrolase] [protein_id=WP_042231399.1] [location=181911..182756] [gbkey=CDS]
MSERMIRVKDIELCTESFGNPAHPAVLLIMGATASMLWWEEDFCRLLAEKGFFVIRYDNRDTGRSASYEPGRPGYTFEDM
ADDAIAVLDAYGVKKAHFAGMSMGGMLVQIIALRHPDRVQSISLVSTMYFGPGWEELPPMEAKVEAFFNSMGEMDWYDPQ
TLLRNAVDKWRILAGSKHPFDEERVTALAERDIGRALNYSSSFNHSYVTGGESYLGRYGEIQTPALVIHGTEDPIIPYVH
GEALRDSIPVAELLTLEGIGHELPPAEWPAIIGGIAKYAAL
>lcl|NZ_CP026520.1_prot_WP_042231401.1_140 [locus_tag=PC41400_RS00765] [protein=polysaccharide deacetylase family protein] [protein_id=WP_042231401.1] [location=complement(182948..183826)] [gbkey=CDS]
MKNIVLAACGIFFGILLILTVGWDRNGRENAAGPVYYENKAIVLLYHDIRKTAGSRIDLPSTVTPGQFEGHLEMLKDRGF
HIVPMEQFVQFMREGGKLPPNAVVLTFDDGYESFYTEALPILKKHGVSASNFIVGISTDLFNRDADPHLNWNQIRELKDA
GMGIYNHTYDLHREVPENGTDIMTPALISRMFLEQKKRSEIDGEYRKRIRSDIAFLEQRLDDETGKHRRLLAFPYGAYTD
AVAVEGRRAGIELFFSIEDGINQPGSEIVRRINAGEPYMTADALWSMLKKVF
>lcl|NZ_CP026520.1_prot_WP_129112438.1_141 [locus_tag=PC41400_RS00770] [protein=hypothetical protein] [protein_id=WP_129112438.1] [location=184166..184345] [gbkey=CDS]
MFTILLVLGIFMLLASFGKALTGKGRSLVQKEQAPKAQFAQDPAPVLALPQPPAEPVGR
>lcl|NZ_CP026520.1_prot_WP_042231405.1_142 [gene=gdhA] [locus_tag=PC41400_RS00775] [protein=NADP-specific glutamate dehydrogenase] [protein_id=WP_042231405.1] [location=184504..185886] [gbkey=CDS]
MGVTLGGKAVNEVAARTYIEEVYQTVLQRNSNQYEFQQAVKEIVESLVPVFAKNPHYMKAGILERLVEPERLVSFRVPWV
DDQGNVRVNRGFRVQFNSAIGPYKGGLRFHPSVNESIIKFLGFEQIFKNSLTGLPIGGGKGGSDFDPKGKSDGEVMRFTQ
SFMTELCKYIGQDTDVPAGDIGVGAREIGYMFGQYKRIRGGFEAGVLTGKGVVYGGSLARPEATGYGTVYFVHEMLQAKG
LSFEGSTVVVSGSGNVAIYAMQKAQQFGAKVVACSDSGGYIYDPNGINVETVRQLKEVERKRISEYTLLHPGSFYHEGCD
KIWTIPCDIALPCATQNEIDITSAKTLIANGVKAIGEGANMPTTLDAIDSFLQAGVLFGPAKAANAGGVAVSALEMSQNS
MRQSWTFEEVDEKLHQIMKNIYQASIQAAAEYDHADNLVVGANIAGFKKVADAMIAQGII
>lcl|NZ_CP026520.1_prot_WP_042231407.1_143 [locus_tag=PC41400_RS00780] [protein=hypothetical protein] [protein_id=WP_042231407.1] [location=186190..186516] [gbkey=CDS]
MSRQNQSLEYTKVAFAGISDAVERTTSRIQVMSRSIDKITEIKDGMLNSISNLSAISQQTAAGTEEVAATMEEQRTAVVA
LTDMSDQLKATAAHLIGETKSFKLEPDS
>lcl|NZ_CP026520.1_prot_WP_042231857.1_144 [locus_tag=PC41400_RS00785] [protein=hypothetical protein] [protein_id=WP_042231857.1] [location=complement(186684..187184)] [gbkey=CDS]
MGTRLLAEQLIQQRYPHLRYVRVHTDGKNTAVIYAWNEELLLTAEDMANLKEFASSYLLPHVCFKVKPYDQLKTDSIPQA
LIQELPETIWKAAVARESNQYRIASALNDLFMRSIRFTFSRYDSVTGTVHFVARASIPVTDEVKERVQRYLYEMLPLGTR
SEVTYY
>lcl|NZ_CP026520.1_prot_WP_042231409.1_145 [locus_tag=PC41400_RS00790] [protein=MgtC/SapB family protein] [protein_id=WP_042231409.1] [location=complement(187345..188013)] [gbkey=CDS]
MVQFDYLLRVLVAGICGALIGYERKSRSKEAGIRTHFVVAVGAALMMTISKYGFQDQLDWPNLGLDPSRVAAQVVSGVGF
LGAGMIFMQKQTVKGLTTAAGIWATAGVGMAVGSGMYGLGIGVTLLILAAQKVLHGRFNWLAAPKSEHLFMRLANEPDAL
ENMLAAIRGRGIKVLSFTMDNATKETDEIVLEITVRLPETYRVEELLNLTREVPALLTIEVQ
>lcl|NZ_CP026520.1_prot_WP_042231415.1_146 [locus_tag=PC41400_RS00795] [protein=uracil/xanthine transporter] [protein_id=WP_042231415.1] [location=complement(188116..189423)] [gbkey=CDS]
MRATLTPKIGLAGIQWMFFILTNTIVVPLTLGHVFQLAPGQIAFSLQVSFVLTGAACMLQAYWGHRFALMDGPAGVWWGF
TLSIAASAAASGADMKAVGGGLAAGYLLSGLMMTVLGLLGFARIMRRLFNPVVISVSLFLLSAQLMASFFRGMLGVGEDG
RMNPPVALLSVLIALFVAVLYLKGRGWVSNFAMLFGILLGWAAYGLAFPAPGSMPDYAAAVQGLFPWGSLRVDAGIMAAS
FIVGFINMSNTVTSLKAAEKLYGEAATESQYRRSFLVTGLMTLLTPVFGLIPFGTFTSSMGLLESMKILHRQALAIGGGL
FLVLGLIPPLGGWFSGLPVSVGSAVLFVAYLQMFGTAVRSVEGTRFNSKTIYRLALPILLGVSIMNTPSVVFSALPVYIR
PLAGNGLVVGILMSLILETCVNWTKLDIAKEKGYD
>lcl|NZ_CP026520.1_prot_WP_042231416.1_147 [locus_tag=PC41400_RS00800] [protein=nucleoside deaminase] [protein_id=WP_042231416.1] [location=complement(189426..189887)] [gbkey=CDS]
MDSTDVFIKRAVELACTNTREKKGKPFGAVIVKDGRIIAEGVNDVLETHDPTAHAEIQAIREASRVLRQSDLSDCELYAS
GEPCPMCLSAVYWAKFKAVYYAYTEEEASRIGMSTKFVYDQLALAPEKRVVQLVRVEPDCPDHPYAVWKESQA
>lcl|NZ_CP026520.1_prot_WP_042231860.1_148 [locus_tag=PC41400_RS00805] [protein=LysR family transcriptional regulator] [protein_id=WP_042231860.1] [location=190019..190897] [gbkey=CDS]
MDLKQLHYFKAIAEEGQITAAAKKLHIAQPPLSQQLKAMEETLGAALFRREGRRMLLTEEGRTLYKHAVKIIASMEEAQE
EIHAVRSGWSGKLCIGVNTLSDDLLPRILGEFRQNYPQFTFKIHQNESSILCRLLKEQTIDLAIVRMPLPLPLQDFTVHY
VRSERYCFVTAAGDMQAQSPVTFEQIREYPLIIPSTEGLGLHHMIHEEFAKRRLEPDIVCECSDMSTLLQLISSGLGAGI
VPETVPKLHPGPLLRYIPIEGDILPAQSALIGLRDGWLSRASRSFIDAYRNW
>lcl|NZ_CP026520.1_prot_WP_241688764.1_149 [locus_tag=PC41400_RS00810] [protein=hypothetical protein] [protein_id=WP_241688764.1] [location=complement(190899..191441)] [gbkey=CDS]
MKTAFETARAGLLQGLPDDEIVRKAKEAARFSGSMRRNLISMLDELELHRYLGLASSGELFTQGIGSVLHTTSLLRYPVF
VQGKNYTGAQPPLLSNEFLRETAFAFTKEELERQPGALIIPLGKTVEGLLLRMEEAGFLGECGILHGFPHPSGANGHRHK
QFAEHREAMKRRLEEKFGRA
>lcl|NZ_CP026520.1_prot_WP_241688765.1_150 [locus_tag=PC41400_RS30150] [protein=hypothetical protein] [protein_id=WP_241688765.1] [location=complement(191471..191644)] [gbkey=CDS]
MLPTSEFHTYKPAILSLPADRPLTKEDLLTGDFLMKRSGRLAMYYAPHNEFVRPPKS
>lcl|NZ_CP026520.1_prot_WP_042231420.1_151 [locus_tag=PC41400_RS00815] [protein=hypothetical protein] [protein_id=WP_042231420.1] [location=complement(191932..192444)] [gbkey=CDS]
MEKQGRFSQKTAVLLERVRRLGLSDEVLLVSAASGDVKPLQEVSGDDSSYEDFFVYGETHGEQIAEGIRNGYRMKFNTTG
GLQSWLKERLGREAETDFAMSVGRIEGLALAADEAEVLRSSLAPNWLMLEVPSAGEGADKAPENEDAPTLPASGETGTYS
LVLKFLSEKE
>lcl|NZ_CP026520.1_prot_WP_129112553.1_152 [locus_tag=PC41400_RS00820] [protein=ABC transporter ATP-binding protein] [protein_id=WP_129112553.1] [location=complement(192529..193491)] [gbkey=CDS]
MLIDVRGLKKYFPASRSLFGRTTRELRAVDDVSFRIHRGETFGLVGESGCGKSTTGRCIIRLHDVTSGEIRYDGRDMAPM
AEKELKPYRKRMQAIFQDPYSSLNPGMNVLQLISEPLEIHSAFSRAERRERVLELLARVGLKSEHADRYPHEFSGGQRQR
ISIARSLAASPEFVLCDEPISALDVSVQAQVVNMLEELQDELGLTYLFIAHDLSMVRHISTRIGVMYLGKLVEVAPSEEL
YTSPAHPYTEALLSAIPVPDPKAAPAGGRILQGDLPSPLESITGCKFRSRCPFAIGRCAVQEPELREIAPGHYAACHLHE
>lcl|NZ_CP026520.1_prot_WP_042231423.1_153 [locus_tag=PC41400_RS00825] [protein=ABC transporter ATP-binding protein] [protein_id=WP_042231423.1] [location=complement(193716..194747)] [gbkey=CDS]
MASLLSVRDLKTSFFTREGEVQSVRGVSFTVEKGEMVGIVGESGSGKSVAAKSLIRLIQPPGKIVGGEVVLQGTDTLRLP
EEELRKLRGNRVAMIFQDPMTSLNPVVKVGTQLTEVIRRHRKLGRKAARLRAIELLREVGIPSPEKRIDQYPHEFSGGMR
QRVMIAMAISCTPELLIADEPTTALDVTIQAQILDLMKSLCDTTGTSVLLITHDLGVVAQVCTRVVVMYGGLVMEEGPVE
AIFETPRHPYTQGLLRSIPKTGGGSREKLVPIEGTPPNLLDPPPGCPFMERCPHAMEKCLRLPPYYEAQPGHRSLCWLAG
GPDNHDSGHVPQAADTKRKEGDS
>lcl|NZ_CP026520.1_prot_WP_042231425.1_154 [locus_tag=PC41400_RS00830] [protein=ABC transporter permease] [protein_id=WP_042231425.1] [location=complement(194773..195618)] [gbkey=CDS]
MNSMRWRLLKAQLLRQKSGLAALILLALYALAAIFAFAVPYDPNAIHLTERLQPPSAEHWFGTDDNGRDYFARALYGGRV
SLSVGFLSMILSVVIGTAVGTVSGYFGGKTDSFIMRGVDILLSIPSFFLMLILNAYLKPSILTIILIIGLLSWMYISRIV
RAETISVKEREYVLYARVSGQSTFRIIARHIIPSIVPTIIVAATISIAGTIMMESALSFLGLGVRPPYASWGSMLNNAKG
YLGENFYLALFPGLFILTTVLSFNYLGDVFRSAFEPKVNRR
>lcl|NZ_CP026520.1_prot_WP_042231428.1_155 [locus_tag=PC41400_RS00835] [protein=ABC transporter permease] [protein_id=WP_042231428.1] [location=complement(195625..196575)] [gbkey=CDS]
MGKYVLHRILQLIPILLFISIVSFALIRLAPGDPVLSFVTPNMSPEDVERIRQNMGLDQPLYIQYGLWLKNVLAGNFGYS
LNNHRPVLTLIVERLPATIGLMGSAFLLSLLVAVVLGLISAVYKNKLADRVLSLLSYIGISLPGLWFALILIYVFTVQLH
WLPSMGMRTIGVHSAWDVIKHGIMPCLVLCITHVSVFMRYIRSSTISQLEEDYVQIQYAYGSTRRTVLFRHILRNVLLPV
ITLFGLSFAEIITGAFIVESIFSWPGMGSLGINAVFGLDYPLIMGMTMMSSFLLIFGNLLADVLYGLADPRIKTMR
>lcl|NZ_CP026520.1_prot_WP_042231429.1_156 [locus_tag=PC41400_RS00840] [protein=ABC transporter substrate-binding protein] [protein_id=WP_042231429.1] [location=complement(196634..198211)] [gbkey=CDS]
MKWSKSLAAVLMSVIVLTASACSGTGSPEGAEGSSAPKTGGGLIVAIKDDPKTFNPLYAGDRMSLTIGKSLFAPLFSIND
GKKTFVLAESLTPSADLKSYVLKLRDGLTWHDGQKLTADDVVFSFEQVLDEKQHSYFRSLFVLDGKPIQAKKLDDLTVEF
TLPQTTAAFEGALVQLIPVPKHIFAGESDLEKSSKNNKPVGSGPFKFKEYRPGEYVTLERSDSYFAGKAKLDSITYRVAK
DSSAAVLALQNGEINMMLIEPQDYNKVKGTGQANVITFPSGGVTTMVFNTNIEVMNKKEVRQAIVHGLDKKELITTGFSS
QEFAEPASSIFSPDTLYQQKDLKGYDYNPEKAKELLKAAGAEGLKIRFAYFNGSKAEENMSLYLQQKLKAIGVTVELIPL
DGAALGKKALNKDNKEYDMSFGGYSMGFEPDAYKPLYLSSGSYNYSHYKNSEFDALWTKAAVETDSAKRAKLYQSIQELS
AEEVTTYPIAYGKSIVAVDARFGGLEEAVPKQVVLFDDYSKIYMK
>lcl|NZ_CP026520.1_prot_WP_042231431.1_157 [locus_tag=PC41400_RS00845] [protein=MarR family transcriptional regulator] [protein_id=WP_042231431.1] [location=complement(198545..198997)] [gbkey=CDS]
MKLTDYCVKVGELLRVIDTKHAKIVNQHYSPHNLTAPSINILLLLDKEGAMRVSDIGSALNMVDSNVSAICSRLENMNLI
ERFRTKEDQRVVKIQLSPEAEDKMKDILANVQEFQELFVKNSSQEELEAIVRGLSLLERLLDNVLKEKMK
>lcl|NZ_CP026520.1_prot_WP_241688766.1_158 [locus_tag=PC41400_RS00850] [protein=PucR family transcriptional regulator] [protein_id=WP_241688766.1] [location=199285..200895] [gbkey=CDS]
MWYDQAMIYLRGEPMTTSFTVKNLLDMPLFRGVEVLAGAQGLSNEIFYVDCMETPDLTGWLRPNELILTTGYSFRFEPSM
LCRLLDEMHRVGGSAVGIKTKRYLQEVHPEAIELSNRYGIPLFDIPLDVPFMDMTRSIMDQILQRQAYLLKEVHEVNQQF
TGMVLNRKFSELVILIGKLLRCEVAILNREGEVESCTADYNAASGVENRPVFTGNRIFGTLSLTRKLEDGDHFEKMCLDH
AVMALAIEFTIRQAQQLHRSREQELFAAELLSGGVSRQEDLLRYRAKHLGIPYGGFVYVMALRASCPADTDSETVKTLHS
RLAEEVNRDSGGQRRLAVEINDRLLLICAADRPETGKQLETARKYKAELEARASRIVGTDAAAMIVRCGIGAVRGQYADL
HASCLEARQALAFGEQAADQPEVLHFHDRLVEHLLMDVAGHPLLEQLHGELIQPLLDYDGEAGTELLPTLEAYLKSGSTK
QVAEELFIHRNSVLYRLERIRELLRTDIHDPDIRFRLDLAIRYGQSRKIKRVCPQL
>lcl|NZ_CP026520.1_prot_WP_174446829.1_159 [locus_tag=PC41400_RS00855] [protein=SDR family oxidoreductase] [protein_id=WP_174446829.1] [location=200911..201684] [gbkey=CDS]
MDAMSTLRDQRVVIIGGSSGIGLAAARQAVGQGAEVIIAGRSAERLQQAEEALKLPESGVSSHEDIHPAGRVRTFRLNNR
NEDELQSFFDRVGAFDHLFTPGASYVRGPITSDAETAHSCFEGKFWPQYNAVKYAVPYLSRSGSVVLMSGAFSQRPLGDG
ASYAACNGAIESLGKALAVELAPVRVNVVSPGTIVTDFNWEGASREKREESYNAYNDMCLLGRTGNADEVAHTVVYLMTN
GYTTGSTLFPDGGYILR
>lcl|NZ_CP026520.1_prot_WP_042231436.1_160 [locus_tag=PC41400_RS00860] [protein=methyl-accepting chemotaxis protein] [protein_id=WP_042231436.1] [location=complement(201805..203568)] [gbkey=CDS]
MKFKQKLALSFAAVLLLTLTVGAIGLYQLTSVNQVYQSLINDRVEKILIIKDLGTTAAEETKNVRGYLLTGNEEHFKSYE
QDRQTFKADMSKLQANIHQDKSKELAARLSSLEEQYAGIVDEIAAYKKAEDIRNYTRLVEEKCVPMARQLADTAAELEDF
QQAQLNQSVQEISHKVSDIKMLLRVTLLTALVAGIVLAYLITRMIARPVTLVAQAAKRIAEGDLTGDDVRLRQKDEIGQM
AGAFNEMKANLRTLLAKINRNAVEVAAASKELSDGSEQAVTGAQQVAESMQDISGTADRQVNQLRENKQAIEESAQGIQR
IAESASITVESSEQAALQAEEGRKLLGETVVQMQQIRTTIEQSAEVIHDLGEQSRNIEKITAFIQDIARQTNLLSLNASI
EAARAGEHGKGFAVVAGEVKKLAEQTGEASEQIAKGIQRMVQTVAQSVTAMNRGSAEVEAGTRVMNRTGEAFNGIYQAIG
AVTGQVQEVSAATEELSAAMEQLLKAEEQLAGLSADISGHSQSVAAVCEEQLASMEEISASSDALSHMAKELQDEIQKFK
FEGDEPGAASQPNAHYRGQGELAASAS
>lcl|NZ_CP026520.1_prot_WP_084706635.1_161 [locus_tag=PC41400_RS00865] [protein=ABC transporter permease] [protein_id=WP_084706635.1] [location=203825..204979] [gbkey=CDS]
MNGFLNFMKSKAVIGGLTMILFYQVAFIAIFMSGYSAIPKNIDELKVAIVNEDTQAGKEIAANLQKELPFRVTTGASLEQ
AREELNDRDVHMVIHIPANFTEKLSQQNQKAPIDFMINQSNPATVTSTMQSVVTQITGSLNQQFAVQSAQGILQNLNMPE
EQAKQLAAEIPTKLAPSTININKVPAGMHNQMAPFFLAMVSYVGAMIFSMMVTGATRAYTPILGRAKAFWYAQGVNALVA
FFSPLIGLTIYFLIQGGYEASVFLQVWLVHALELFAAIEFMSIFCYLLKDAAIFANLTLMLFQTISSGAVMTQDMMPALF
RWISHISIMYYTVQTDFSLLFGGGKIGENLLGLAGMAIASLVVGQLAYLFVSPGFKKKALTDAS
>lcl|NZ_CP026520.1_prot_WP_042231439.1_162 [locus_tag=PC41400_RS00870] [protein=PadR family transcriptional regulator] [protein_id=WP_042231439.1] [location=205064..205594] [gbkey=CDS]
MSLQIFILGILCEGNHHPYDIKKMFKKNNMEDVHKISDGTLYYNFDVLLKKGCIEKLEVIRDDNRPEKTTYGITPKGREF
LEQEIYASFRNFTGVESLYSPTLFLKHADKNKLVFLIEEVMDKLRNRTEQQNQEWDAFKDKTPPSVHLIQEYVQSRMKLE
SEWLEKLLAFVKNGSE
>lcl|NZ_CP026520.1_prot_WP_042231441.1_163 [gene=yedE] [locus_tag=PC41400_RS00875] [protein=selenium metabolism membrane protein YedE/FdhT] [protein_id=WP_042231441.1] [location=205781..206995] [gbkey=CDS]
MRETVKKIFTHYWSPYIAMGLAGVLSALYFGITNTVWAVTGEFTRLGGHILQLFGVDVSDWAYFKLVGMNGTTLTRTDGW
IVWGMFIGALVMILLSNNFKIRIPKQKRRLVQGFAGGIIAGFGARLALGCNLAAFFTGVPQFSLHSWIFIAATAIGTYLG
VKVVNTAWWKGKPMLSKGNTAAAPSAAKKRIVQPYVGAGIALAYAALIVYFFAAGKTMLGAAAIFGAAFGILIERGQICF
TSAFRDLWISGRATMTKAITAGMAISSVATLIIILVYGLEPITKIAAPSTLVGGILFGLGIVLAGGCETGMMYRLMEGQV
VFLPVFIGNIIGATALAYAWDHLGVFNLLAKSGSKINLIPLMGPYGALFATLALLAAGFAVAAYWQKNYRFGTGFKKGDT
THVR
>lcl|NZ_CP026520.1_prot_WP_009675875.1_164 [gene=yedF] [locus_tag=PC41400_RS00880] [protein=sulfurtransferase-like selenium metabolism protein YedF] [protein_id=WP_009675875.1] [location=206985..207212] [gbkey=CDS]
MSVDYTLDIRGESCPYPVIYTLETLRDMKKGQLLQVIADCPAAFRNVPEEVVKHGYTMAQEPVRDGQEFTLYIRA
>lcl|NZ_CP026520.1_prot_WP_042231444.1_165 [locus_tag=PC41400_RS00885] [protein=DUF5050 domain-containing protein] [protein_id=WP_042231444.1] [location=207376..208776] [gbkey=CDS]
MRLVPALIGLSLFDFSAGISSASASPGQDVSVVVNGAELFPDSRAYIQGESGLTYVPLRSLSEAFGTEVEWSQSTKTATV
RKGATLIDLTVGSTLAHIDGREAALESPAEQENGRVMVPVRFIADALGVAIAWNADTRTVTLGDTPAPASVFAPGEGNDV
QNILNQAFVTQSNGWVYYQNVGDESKLYRMKEDGSQPVKITDDPAVQIQVTGGWVYYRLLTQGERTDAGEVYRIRPDGSE
RTKLSKEKAGQVYVTGDWIYYNVTAGFGSGPLFKMKLDGSGKQELSQKDEDVMDFVLYQGRIYYKRSTGGLYSLDLDGRN
RKTYTKEFVHELRAANGSIYYHKDANQLLKMSIQTGEESVAVDNVPLSRYYVGPDYLVLEKALSFLDRKVYLTDLQGAGE
KLRVDNTLVQLNVIGDWIYFITYETKSSETHRYADIVISLGRIKKDGSQREIITSKTSNPFPNDRL
>lcl|NZ_CP026520.1_prot_WP_129112439.1_166 [locus_tag=PC41400_RS00890] [protein=hypothetical protein] [protein_id=WP_129112439.1] [location=208916..209374] [gbkey=CDS]
MRKRYAKNLLALGVVASALLVGSQSAFASDGVGDSRGTAIPISPNSAAVQTISDSTDVDYFTWTNNTGNRISSASVHLYS
PSNNNYDLQVIQILSQHNNFEATTTASDHGNGIEDSVSFSPIYPGDTVYFKVYGHDSSQYGSAPYTLSFFTY
>lcl|NZ_CP026520.1_prot_WP_129112440.1_167 [locus_tag=PC41400_RS00895] [protein=NADP-dependent oxidoreductase] [protein_id=WP_129112440.1] [location=complement(209709..210728)] [gbkey=CDS]
MPMNKQILLRSRPEGMPTVDHFEFREVPVSDPAEGQVVVRSLYLSVDPYMRGRMNESKSYVPPYKLDEPISGGIVGEVVS
SRSEQFETGDKVLGMLGWQLYNTVDASAVRKVDEDLAPLSAYLSVLGLTGMTAYFGLLDIGKPQEGETVVVSGAAGAVGM
MVGQIAKLKGARVVGIAGSDEKTAYLEKELGFDAAVNYKTADNLDEAIAKACPDGVDVYYDNVGGRISDTVMKQLNDHAR
IPLCGAISSYNSTDGDLGPRIQPQLIKTRSLIQGFVLSDYASRFHEAGADLGKWLSEGKLKYEETIVEGFDNVPKAFLEL
FKGTNLGKMLVKVSEADGE
>lcl|NZ_CP026520.1_prot_WP_042231448.1_168 [locus_tag=PC41400_RS00900] [protein=metalloregulator ArsR/SmtB family transcription factor] [protein_id=WP_042231448.1] [location=complement(210896..211189)] [gbkey=CDS]
MTIEQLIVEDEQRVKIFKALAEVKRIEMIRYLHRHDRSNTCGEIGESIGMDKSNVSYHLKILYEAGLIKVSRSGQNKTSE
LCRDVFYRFLPGFLDSL
>lcl|NZ_CP026520.1_prot_WP_042231449.1_169 [locus_tag=PC41400_RS00905] [protein=helix-turn-helix domain-containing protein] [protein_id=WP_042231449.1] [location=complement(211221..211622)] [gbkey=CDS]
MTTKKKYNIPVEATLEIIGGKWKCVILCHLTHGKRRTSELKRLMPTITQKMLTQQLRELEEDGVIKRTIHQEIPPRVDYE
LSEYGWGLKKILDSLCAWGENHIVKVYGDKSAVLEDSFLNDKSFDKEEESPVN
>lcl|NZ_CP026520.1_prot_WP_042231451.1_170 [locus_tag=PC41400_RS00910] [protein=MFS transporter] [protein_id=WP_042231451.1] [location=211821..213023] [gbkey=CDS]
MIAVSQKSTLALLALAISAFAIGTTEFISVGLLPLIAEDLQTSLTTAGLTVSMYALGITLGAPILTSLTSTMTRKTLLLW
IMVVFIIGNSLAAGATTIGVLLAARIVSSLAHGVFMSIGSTIAADLVPEHRRASAISIMFTGLTVATITGVPIGTFIGQQ
WGWRLAFAAIVVVGVVAFIANSVLIPSTLPKGVRTTFRDQLKLVTNGRLLLMFVITALGYGGTFVVFTFLSPLLHTVTGF
GENTVTLLLLVYGVAIAVGNVIGGKAANRKPLPALLVMFLLQAVVLFTLAFTAPYKAAGLITIILMGLFAFMNVAGLQVF
VLTLAKRYVPAAVDVASAINIAAFNAGIAIGAYLGGIVTERMGLIHTTWVGGIMVVAAVVLTGWSLLLEGRNAAQAKAAS
>lcl|NZ_CP026520.1_prot_WP_042231455.1_171 [locus_tag=PC41400_RS00915] [protein=diguanylate cyclase] [protein_id=WP_042231455.1] [location=complement(213127..213981)] [gbkey=CDS]
MKVNRSSLLSDIAFLVLLVLCNICVAFTAGDPNKYVENLIFLNLAFFFVIVTYFTTVTAGLILNIVFIFGYGTYILYETV
TRGETIGVSTYFWLIMTPVLTVVIWLFTTATRQLQAENDDLQKQKARLATLDERTDLKNSLAFRKDANVFAGISTRYNIP
LTLLVIKVKYWNEIRPLIGDEQLAEAIIDVSKISQASIRTNDALYLLDKEDATWGLLLFTDRDGVKVVMERIKQSLEEFN
TREFAQKYKVDLSLKMGAIQYQAETIHSPIDFIVQAKKQLEYDV
>lcl|NZ_CP026520.1_prot_WP_042231457.1_172 [locus_tag=PC41400_RS00920] [protein=cellulose biosynthesis cyclic di-GMP-binding regulatory protein BcsB] [protein_id=WP_042231457.1] [location=complement(213971..216061)] [gbkey=CDS]
MMTMKKWLVFICLLSIFAVQFGQVAAAAPASGAARTYQTAFTESEASLSGAVSSKQKYFELMDYWNVDTVKINLDYQATQ
LAKELISSVTLSVNGHAFYSFRPVAADNGRQRLSVSAPKEWLKKGSNVLEITGYIRTVDNEICAPDDAPDHWLSLFTTSS
IDVAYTNAELKAAINDFSHRFAGLDTVGAGQSVLTVPDNGDSAELEAAVYALSGFAKAKKNADKDIPLLPYRADVLEDKQ
AVVLVGLYDRLPEGIKGLLSKQDYNTKAVIQLVNKDTQPTLVVTSQNPKLLVHAGRFVANPSLMEQTDSDLKIVEESTDV
TTPPVAYSTQVTFTETGDEVRGPGHREKSYFVSLPTNRSIADAGQITLDFRYAQNLDFNRSLVTVSINNIPIGSKKLTKE
MANGDKAVFAIPSNAAVSGNFSVTVAFDLQMISAACTQNGNQTPWAFLTKDSMINFKTKDRTDMLFNNYPYPFLRDGVYN
RVAVVLPEAKDDYTYRSVSNVFNLLGQYASGNTGDVRFYKDTAGSDELKDSNIIAIGTFIDNKLIQENNKNLYFKYDPKG
EGILSNEKISIETEYGKRLGTLQLLASPFQAGNGFLAVTGARAENYYLASKLIANENAKWKVYGDGVFTDKDGSVSAYRF
KKEAGEGDQSLVKDVLQRGDVLGFLVAFVLIAALAVLSLILLIRKHLKKRGGRDES
>lcl|NZ_CP026520.1_prot_WP_042231458.1_173 [locus_tag=PC41400_RS00925] [protein=glycosyltransferase] [protein_id=WP_042231458.1] [location=complement(216108..217361)] [gbkey=CDS]
MTISDYLMLIAVLCIWSLLLVNVLLIIAGYFYYVQTEREPMPDIPAGEEPFVTIMVPAHNEGKVISQTVEALLAFDYPHD
RYEVIVINDNSSDNSAELLAGIQARNPRRNLIIINTDSVTGGKGKSNALNLGFAQSKGELIAIYDADNTPERKALRYLVA
EITNDASLGAVIGKFRTRNRDVNLLTRFINIETLSFQWMAQAGRWKLFKLCTIPGTNFIMRRHIVESIGGWDVKAIAEDT
EISFRIYMMGYRIKFQPKAVTWEQEPQTVKVWFKQRTRWAKGNIYVIVKNCKLLFRPEAKRIRFDIIYLLSIYFLLVTAL
VTSDVLLVLNALGYVHTTIASFSTFLWLLAVTLFVVGTYVTLTTEKGEMNLSNVWIIMLMYVSYCQMWMAVAAYGLYNYF
KDQLFGREVKWYKTERY
>lcl|NZ_CP026520.1_prot_WP_042231460.1_174 [locus_tag=PC41400_RS00930] [protein=hypothetical protein] [protein_id=WP_042231460.1] [location=complement(217372..218922)] [gbkey=CDS]
MRSIQAYKISLPRLFLLVLLLIPLPHASAAGSQPVKVLLLYDSLAKGTPREGNVTVLQQLLASYRTQVTVSSIDRYKQGT
MQAYTNVVTVVNDGDIRISNRAYLDDLSAYPGAYLHVGYQLPAQTRDKLSAATETVSRDTAEIAADKRTQADIPVRDIEV
LTQTGGSTYGTITLSGSGRKAPFSAEAGGTAYVPYFQKGNLSELAMATVLKKWLGIEENGQAYVVLKEIYPFSDLNLLEE
TADRLYGAGIPFVISIRPVFGNTDFPAMKRYEEALKYVQTRNGSILVNAPVVLDGVSREGGTLREKMNWFIAVLAKNGVA
PLGVGTEMYWAYDKEYAKSGLGFFDSAVLFPDVKTLYIDPSDTSKPFASAPYSMPMSFLRQFGAADRSVPVFPMDTAVVY
DFFGSKEEMEEAVGTLGSSGIPFADYKYGSHEVKAGDTLVASSRGTVTINGSPVNTEYTPQKISADYEYEKKQQQSLAGA
FSVQNRIFIIIIVVSLVVFGGLIVVGYRLYRKKYLK
>lcl|NZ_CP026520.1_prot_WP_042231462.1_175 [locus_tag=PC41400_RS00935] [protein=membrane protein] [protein_id=WP_042231462.1] [location=complement(218900..220462)] [gbkey=CDS]
MTTEKSIKERLRLGFLLAGRYVSPALLAALGVLLITGFALFVPPYIGVADNGDFFRSIYSEGLYFNLPDYDSQRFGYFVK
QYGIYQYFNENGTTIVSSQSLFIKAALLLNQLLFSSVVFDIRFQALLFTLLLTAAVYLLVEALTWKIARKHGYLIALLAI
FMFGDTGYTAYFSSFFGESIVYVMMLFVFASWLLMYRKRYNDYALLALFVISTLILTTSKQQNAPVGMIVACMAVVLLWV
RKEKLFRLITAGSLVLLFVSGVATYAMISKEFVNINQYHAMTRGVLMQTANPENTLKSFEIDKQYAILKGSIFYEPYATV
DVNSPMLEKDFYSRYGFVSILAYYVTHPDKLHSILNVAAKDAFTIRPTAMGNYEASAGKPFGTHADFFSGYSVMKKGLSP
KTYGFIVIWIVVIVGLYMPSFVAAFRARDPRLAQRLLLIVTMIFIGLAGILVSIIGAGDADLAKHEFLFTVAFDLVTFMA
IADGLSRRLMRNHEYAADEADKIRTGTGKGGNALAQHSSL
>lcl|NZ_CP026520.1_prot_WP_042231468.1_176 [locus_tag=PC41400_RS00940] [protein=hypothetical protein] [protein_id=WP_042231468.1] [location=complement(220485..221615)] [gbkey=CDS]
MKKGTAIALGVVLCLVLFLIVNRKTEDAPPMPAPSGEPAGQNGKIINVQNERQALYAFVKTRMTGPFGVFTNFKETDEAK
EAATGHEVLSESASLQMREDARSKRKEEFARHWALAQQTFDLKRGFSYRYSPKQQKKYPLNAAVDDLRVIRALAEAAKAF
DEPAYAKEAEKYGKRFYETNVKDGRLYDFYDETYGITNKFVTLCYIDLRTLGMLPAPDASRQALLDNMKSLIAKGYLSDA
FPFYETRFDYEKNAYTSEGINMVESLLTVLHLAEAGEPAPASIRYIKDHVKAGTLYGRYTKEGTPATQVQSTAIYAITAM
IGSEIGDASLYQDSIQRMNAYQVKDEGSPLNGGFGDVASGQAYSFDNLMALLAYTY
>lcl|NZ_CP026520.1_prot_WP_042231470.1_177 [locus_tag=PC41400_RS00945] [protein=family 20 glycosylhydrolase] [protein_id=WP_042231470.1] [location=complement(221927..223837)] [gbkey=CDS]
MKLRFEGDLTGLQPGLQVLLQELSVTEAEDGLKVAVGPSSGDLEVGLEQGAAYIRYGQKHQFYRALGLLVQHGSGGESFS
IREKQHLDVVGPMFDLSRNGVLTLESFKFMIRKMALMGMNTVMLYLEDTYEIEGEPYFGYMRGRYTQAELKEIDDYADQF
GIEAFPSIQTLAHLEEFLKWEPVKDYKDTKGALLVSYDKTDELVEKMIDAASAPFRSRKIHIGMDEAEELGRGKYLDKNG
FTSRFDIMITHLDKVLDMTRRRGLKAMMWSDMFLKLASGTGSEHYNKDTQIPAEMAARIPKDVDLVYWDYVHTEETDYEN
QIAQHAPLGCNLVYAGGVWVFNTFGVNYGLSLKATDAALRVNKKHGIREAYATMWGDDGMESNPYIALLGLQFYAEHAYT
EGSPDWDVLAERVKFCTGLEADAYLQMKYLDETPGSEPDNRKQSNPSKFLLYQDVLLGLFDKQIEGLEMSAHYAKVEQDI
SSRRDPGASLDYIFDVPEKLAGVLKRKSEIGIELKRAYDAGDKDTLTRIAKDVLPEIAEAVRALRAAHRTQWLSMFKPFG
WEVIDIRYGGVLSRLDTASVRLLDYAEGRIGRIEELEQERLLYSTSNRFTNKGVGWCSFYYRMASPNVFFHVNNPF
>lcl|NZ_CP026520.1_prot_WP_042231472.1_178 [locus_tag=PC41400_RS00950] [protein=response regulator transcription factor] [protein_id=WP_042231472.1] [location=224344..225021] [gbkey=CDS]
MKKVLIIEDEAPIARILQAYLEREPYSVKWNPGDGGMLDLFLTWKPDLVLLDLMLPDLDGLDVLKQIRQHGSCPVIILTA
RGSVPDRLQGLQQGADDYIPKPFDPDEVIARVQAVLRRSAYMADSRTVRLGSLVVDFTACTVYLHDKLLPLYPRDWSLLA
FLIKHPNQCFSREQLLDRVWGMDYEGGDRVVDTTIKRLRQCLQDWPSSEGEILTIRGLGYSLHVH
>lcl|NZ_CP026520.1_prot_WP_042231475.1_179 [locus_tag=PC41400_RS00955] [protein=HAMP domain-containing sensor histidine kinase] [protein_id=WP_042231475.1] [location=225011..226510] [gbkey=CDS]
MFTKPQTQPQTMPMLRFWTWRYALILFITLLIVSLIAGVWITSNANRQSYEVLQARAEQLADSYARVMTLYPLSGPAQTQ
GSIRTEVRANPAVKVSPSTTSFNVISPIEVARPLGDEMMQIVNRQGNLDTFSSSGAQYDNAERPGGYAAVLKGQTDRESF
SLNDRTWLRVGVPLMDRGSVIGALYLSTPTPDNAEQTRQTYVIAVLITLGLALGGWTAIYLLLRKLTRPLRQLANAALQV
ADGHYAPVLPVAPMKERELMQLFTSFNEMTSRLRQLEQMRTDLLAGVSHELRTPLTSIRGMVQAVHGKVVTGGEANEFLE
ISLEETKRMQKMIDDMLEFSSLEAGAVHSEKTTVDLSLLVTHLVQQLSSLPACANVRVETVLPAETIRLLGDQGQLKQML
MNLIMNSTAAGSTSITVHAYPLGSMIHVDVKDNGSGIRESEVPHIFERYYRGDSRRKKKQGLGLGLPLSRLLAQANGGEL
ALQETSPKGTVFKLTLPAV
>lcl|NZ_CP026520.1_prot_WP_042231478.1_180 [locus_tag=PC41400_RS00960] [protein=hypothetical protein] [protein_id=WP_042231478.1] [location=226781..227539] [gbkey=CDS]
MKKKLAMGTMALAVFAMAGSAYAAQPEASAPAFTKAATAISAVKGSTAVFNEVKLPQGVSFEEFAKKQGLTAEQLIEKLK
KDGTLDHLAEAKGQTVEEAAEFLKKNGLFAASHNALAISTFSPAEQGDGKTDGTAGNTIKAVLLEKNVTNDKGQTAETVE
DIALPPSLSFEEFAKAQGLTVDELIARLKEDGTLGTIAKANGQSVEETIEFVKKNGLFAAAGKASLFSTIISSVDGGTVA
RGAVQAIKLDSK
>lcl|NZ_CP026520.1_prot_WP_042231480.1_181 [locus_tag=PC41400_RS00965] [protein=hypothetical protein] [protein_id=WP_042231480.1] [location=complement(227831..228466)] [gbkey=CDS]
MRLRLLHLILGDDQYSLPVSDPVLRSKLAKMDSAVSSCAAASAPCSSGKGVWDILNRGADRTYLWVDLPFTSTDSYTKSG
DITNSYNRLLSMITVYYTQGSSFYGNTALKNDVLSALDWLYTQKYNTTVAPYGNWYHWQIGVPQALTKIGILMRDDLTAT
QLTNYMSAVNRFIPDSAARTVSGSPLMTGANMLDQAISVAYRGILVKDPRK
>lcl|NZ_CP026520.1_prot_WP_042231483.1_182 [locus_tag=PC41400_RS00970] [protein=peptide MFS transporter] [protein_id=WP_042231483.1] [location=228894..230381] [gbkey=CDS]
MPAIDKQKIVESVPQKGFFGHPKGLFTLFFTEFWERFSYYGMKAILVFYMYYEVTRGGLGFPEGTALAIVSIYGSLVYMS
GIIGGWLADRLLGTSRAVFYGGILIMFGHLVLAVPGNVALFFVSMVLIVLGTGLLKPNVSSVVGEIYSEQDTRRDSGFSI
FYMGINLGGFLSPLIVGTVGMKNFHLGFGLAAIGMFLGLVLFVSTKKKNLGLAGTVVANPMSPAEKKKVFTIMGIGIVVA
AILVAIAIPAGLLTFSTFIKLIGVLGLLIPTIYFIVMYRSPKTTAVERSRLVAYIPLFIAAIMFWAIQEQGSTILANYAD
KRTQLDFMGLNISPAWFQSLNPLFIIIFAPIFAWLWMRLGNRQPTIPKKFSLGLLFAGLSFLVILLPAYFGGGALVNPLW
LVLSYFIVVVGELCLSPVGLSATTKLAPAAFTAQTMSLWFLSNAAAQAINAQIVKFYTPETEMLYFGVIGGAAIVLSLLL
YVLSPKIQSYMKGIK
>lcl|NZ_CP026520.1_prot_WP_042231485.1_183 [locus_tag=PC41400_RS00975] [protein=metalloregulator ArsR/SmtB family transcription factor] [protein_id=WP_042231485.1] [location=230618..230887] [gbkey=CDS]
MTDDYAVKVYKALGEITRLQIVKALVKQRELACLEMAELLKITAASTLTHHLKPLLESGVIAVRKEGTFRYYRIQRHVLE
QYAPYLLGE
>lcl|NZ_CP026520.1_prot_WP_042231487.1_184 [locus_tag=PC41400_RS00980] [protein=alkene reductase] [protein_id=WP_042231487.1] [location=230955..232070] [gbkey=CDS]
MAENKELFEATGISTWRKLQDGPESKLFEPVKIGAWSLRTRTAMAPMTRCFADDKTGIVGDDVARYYRKRAADGIGLIIS
EGIVVSPRGKGNPGVPGLYTQEQIEGWKKVTEAVHAEGGTIIAQIWHVGRLSHHELTGNLPPQAPSAIRAEGHVARFRKP
YDTPEEMTAADIQELAGQFAQAARNAIEAGFDGVEIHGAHGYLIDQFNSDASNRRTDRYGGDLAQRLTLTKEVLKAVIDA
IGSDRTLIRFSALKVDDPDYMWPDPDGAIRTFTEAFREAGATMIHPSTMHFTRILADGKTMHQLVRKYWDGVIVGVGTLN
PETAEQALAEGTIDVAAFGRPLLSNPDFIHRVKNGEPLEEYDAKKHHGVLV
>lcl|NZ_CP026520.1_prot_WP_042231492.1_185 [locus_tag=PC41400_RS00985] [protein=SDR family NAD(P)-dependent oxidoreductase] [protein_id=WP_042231492.1] [location=complement(232220..233038)] [gbkey=CDS]
MGRVQGKVALVTGGASGIGLSAAALLAREGAKVVIGDFNVEGAREAAETIKNEGGEAAGVFLDASSEQSIRDAVDFTVEQ
YGTLNVLFNNVGLTNLKKDVDVVNMDLEEWDRLMNVNTKSVLLGSRFAIPHMIKAGGGSIINTASMAAFASDATRSAYGA
SKAAVVNLTRYIATQYGKDNIRCNGVAPGLILTPAAKNNMPEPVLDIFAKYNALPYHGEADDIGYTVLFLASDESKFITG
QTIQVEGGHYLANPTVPDFKAFMAQAAGASQA
>lcl|NZ_CP026520.1_prot_WP_009675904.1_186 [locus_tag=PC41400_RS00990] [protein=MarR family transcriptional regulator] [protein_id=WP_009675904.1] [location=complement(233063..233494)] [gbkey=CDS]
MDKNDIFKLIHTVELFTNEAIIRWTKSFEPNIGISPIITLSELKQNGPQKQSVLASKLGYTPGAMTNIAGRLTKLGLAER
RYNENDRRNVFLAITDQGITVLREAQQKGKELRIELFQVLNEEEIQQFLSIYEKLLKNIKEQE
>lcl|NZ_CP026520.1_prot_PC41400_RS00995_187 [locus_tag=PC41400_RS00995] [protein=SDR family oxidoreductase] [pseudo=true] [partial=5'] [location=complement(233569..>233703)] [gbkey=CDS]
KIPMRRLGGIENDIGRMALFLASEDSAYMTGQTVMVDGGATKLR
>lcl|NZ_CP026520.1_prot_WP_042231498.1_188 [locus_tag=PC41400_RS01000] [protein=alanine/glycine:cation symporter family protein] [protein_id=WP_042231498.1] [location=complement(233967..235397)] [gbkey=CDS]
MIDFLQGMINLLNDLLWSKLLIVLLIGIGLYFTYRSKFMQFRMIREMLRVLFESKSVSGDNISPFQAFCISMAARVGTGN
ITGVAIAVALGGPGAVFWMWVIALIGSASSFVESTLAQVYKIKDKNGFRGGPAYYMQTGLKKRWMGALFAVLITISFGLV
FNAVQSNTITIAFENSFGTDRLTVGIIMTVIFGAIIFGGVKRIAQMSEYIVIVLAVLYIGVAVFVVLMNITKMPEVLALI
VKNAFGIDQIAGGSLGAALLQGVKRGLFSNEAGMGSAPNAAATATTSHPAKQGLIQSFGVLTDTLIICTSTAFIVLLSGA
YTDPSLNGIELTQAALSVHIGSWASGFLAIMVFLFAFSTLLGNYYYGETNIEFLHSNKAWVRLYRIAVLAMVIFGSVAKV
QVVWDLADLFMGLMVVVNLIAITWLSKIAFAVLNDYVKQKKAGKDPVFTKDRIPGLEEIECWESAPAASGSDQARS
>lcl|NZ_CP026520.1_prot_WP_164977105.1_189 [locus_tag=PC41400_RS29615] [protein=hypothetical protein] [protein_id=WP_164977105.1] [location=complement(235512..235658)] [gbkey=CDS]
MMRKMVLTWLLFTLFLAAESAEAKMNVYSPPPQPDQSDSGVPGPPTEP
>lcl|NZ_CP026520.1_prot_WP_241688767.1_190 [locus_tag=PC41400_RS01005] [protein=methyltransferase domain-containing protein] [protein_id=WP_241688767.1] [location=complement(235780..236586)] [gbkey=CDS]
MTITEGKSLRCGVGHNFDLAREGYLNLLPQAVKTKYGKNLFAARKTVNENGLFDPVRDYIGERITEAVLPDKRHPKLRIL
DAGCGEGSLLSGIRRRLSDRAATDLLAVGMDLAKEGVRQASKGDAGALWCAADLANAPFKTAGFDVILNILSPSNYAEFR
RLLDEDGLLIKVIPEENYLREFRQLLYAGTEREFFTSGLAVERFRERFRLLDSRRLRYSWRFGPELLEPLLTMTPLAWGA
RDERLREATAMAIDEVTVDVTVLWGKPL
>lcl|NZ_CP026520.1_prot_WP_042231501.1_191 [locus_tag=PC41400_RS01010] [protein=MarR family transcriptional regulator] [protein_id=WP_042231501.1] [location=236922..237386] [gbkey=CDS]
MEKNRTSQTNNSTGANSGGRSLGRLILQLRRLERRPQTFGSAGPLTPSEIHTIEAIGTEGGILMSELAGRLEITKGAVTQ
LIVRLEAKDLVARSPHPEDSRGTLISLTAKGKEAHTAHEEVHRAFYEQLQEQLTEQEIEVFDTCIDKLVTFLKK
>lcl|NZ_CP026520.1_prot_WP_042231504.1_192 [locus_tag=PC41400_RS01015] [protein=MFS transporter] [protein_id=WP_042231504.1] [location=237471..238925] [gbkey=CDS]
MQAARTRTKELRPSLMVTAICLGAFISHFTAGIVNVSLPQFAAIFGTTLGTAQWITTGYLLVIASLLPVMGKLGDRWGFR
LIHNLGYILFGISSVLVAFSPNIGALLGFRIVQAVGAAMFQATNMALITLHLPKQNRGRALGTVNTAVALGAMSGPIAGG
MVAEWLNWHWLFLIHVPVAALATLLAFRFIPPAPVMDKQRSAPFDTMGGLMFMAIISSVIWTISGSSSRGWLSPVTAAVL
ASAALLLAVFMIWERRQANPFIPVRAFRIPALSAGLIISCVAFVLANSFLVTLPFYLSGVSGMSPSAAGAVMTAYPILLA
VTGPFAGHLSDRFGSGLLVLLGLLAMGGGAVVFSLGLGVLPMAGLIALLALVGLGMGLITSPNNSTILHHAPKEHVGAIG
GMIALTRNLGMVVGAALGLGVVQGAAPSASSEPDRLLGALSSVFEINVWLCAGAIALLGLSFYLEKRRRVQLDPVRSAQP
ETHP
>lcl|NZ_CP026520.1_prot_WP_241688768.1_193 [locus_tag=PC41400_RS01020] [protein=alpha/beta hydrolase] [protein_id=WP_241688768.1] [location=complement(238936..240459)] [gbkey=CDS]
MHFEDGVTVHYHEYPPEEGNGQTLVFLHGLGLDRTIWNELIPYFTGHYRIVLYDLPGHGLSTELKENSLDWDFITADLRR
VLKALEIASAHLIGHGIGASAAVQFCRSYPECVDRLVLISLPVKIPADLIRRKITERKRLIKEGVDALAAELAAFKTSLP
VHTSAFQIVREAYRRMNPDVYVGFMELFMEGDGWKELKFIAHRTLVLAGECDPIYVTAASLAVGVLPSASFLIVPGAANM
PFVEQAGTTALWIEQFLHRKDSRPLADEPLVRELRDTLQLFLEEGEKQAALAQVLKIDLLGPFRVFLNGREMVDGWNRRY
AKSLLVYMIFHPTATREELCDALFPQISGKAALRNLKVYLNHFKNMLQSFEGGSSIVSVDREYVRLLCLLRCDLLDFLKE
LEEIRTIDENRRYEPAGRLLARLPDPLLPGLYDEWFVAKKNWLEMEVADLASWMAEKEEQRGSRQRAAAYRQMAARFLFD
REPEYREGAAVNGRRGKTAEERRRKNG
>lcl|NZ_CP026520.1_prot_WP_053228748.1_194 [locus_tag=PC41400_RS01025] [protein=C39 family peptidase] [protein_id=WP_053228748.1] [location=240722..242176] [gbkey=CDS]
MKKHTRPQKGSKRIGWVLLLSVSVQVLLPTQPLFAAKSEKEIVLRIDSSLASIDRKLVSISPEDATAVPFLEQDTTFVPL
RFVSEALGAKVEWEQATQTVTMATYDRQILFAVGSRQITVNGQDHVLSVHSILRNNVTFVPLRGIAESLGSHVDYEDGFI
RISGQQTSSSSSGTAPEAPIEKNTALKEGMSYRVYQHANFLDGFATRQEAVSFAKQYANSTVKNRKDEVVWDSFPPYRVY
QSEKFLAYYDSYTDAVSHAKKYANSFVTYKTAAKPVWTFGGPSKQPVYINVPLIRQLPELPRGCEVTALASLLQFAGVPV
DKMTLAAEIRKDPTPYSEKNGVISFGSPHRGFVGDMYSFNNPGFGVYHGPVFELAERYLPERVVDLTGTEFEDVLHFVGT
GTPVWIISNVTFRKLEPYHFQTWMTPDGPIWLTKKEHSVVIVGYDEDYVYIRDPHNKTGRVAKQPFKEAWEQMGRQAISL
LPEV
>lcl|NZ_CP026520.1_prot_WP_042231508.1_195 [locus_tag=PC41400_RS01030] [protein=M14 family metallocarboxypeptidase] [protein_id=WP_042231508.1] [location=242224..243624] [gbkey=CDS]
MPNKIRGMLLVVFALVPFLFAAAAHAAETTDIVNPIQTYTYEQMTEDIRQLAKSYPELIEYKSIGKTRYGREIWAVRLGT
GDAKVLVNASHHAREWMTTNLVMNMIDQYAQLYVRNGAMNGRALQSVLNNTSIWFVPMVNPDGVTLQQKGLSAFPSEARA
GLLAMNEKSSDFSRWKANVEGIDLNRQYPAEWKSIRNDSGRPYYKNFKGDEPLQTAETKAMVAFTKEVDPDMELSYHSSG
SILYWNFHTLDKNLERDRKIMAQVAAMTGYTPVKPESNPSGGGFTDWFIQSLGRPGLTPEIGRYAGEGSLPLSEFPRIWI
ENRDLGLFAAQQGFELWKERIPVEYKSFDLILTEPLPLYSEADSRNKTGASLTAQPLKADARLGNWFRVDTWLGKSWIHV
NKPLLGKIEVKESKVELTGKTLMYREPRLSSGILSELSPQTVTAFEQMEDWYHIRTWLGDAWIQVK
>lcl|NZ_CP026520.1_prot_WP_042231511.1_196 [locus_tag=PC41400_RS01035] [protein=DUF1003 domain-containing protein] [protein_id=WP_042231511.1] [location=complement(243692..244255)] [gbkey=CDS]
MEKKPYFQWKKEPEDASKIQGFDIELNQESTKRIDRLVDHYEKSILSHLDEEYVKKTRWSDRIADRLAFFGGSWKFIIYF
ALFLAVWITWNTLPFTPLHFDQAPYILLNLCLSFIAAFQAPVIMMSQNRQAARDKHESVIDFAINYKAEQEIDAMQSRLH
RIENELTEVKQLLTALASQKNNGDGAE
>lcl|NZ_CP026520.1_prot_WP_042231513.1_197 [locus_tag=PC41400_RS01040] [protein=LLM class flavin-dependent oxidoreductase] [protein_id=WP_042231513.1] [location=complement(244586..245647)] [gbkey=CDS]
MEIGISTFVETTPDVHTGEVISHAQRLREVVEEIVLADRVGLDVYGVGEHHRKDYAASSPAIVLAAAAPQTSRIRLTSAV
TVLSSADPVRVFQDFATLDGISGGRAEIMAGRGSFIESFPLFGYDLDHYDELFDEKLELLLKIRASEQVTWQGGHRPAID
NLGVYPRPVQNPLPVWIGSGGNQESVVRAGLLGLPLVLAIIGGSPLQFAPLVQLYKKAAAHAGHDASKLTVASHSHGFIA
ETTELAADKFFPSTQAVMNVLGRERGWGRYDRSSFDSARSFNGALYVGDPETVARKIVHLRKNVGITRFLLHVPVGSMPH
DEVMRAIELLGTEVAPRVREEVAAWEAEGADRA
>lcl|NZ_CP026520.1_prot_WP_042231515.1_198 [locus_tag=PC41400_RS01045] [protein=GNAT family N-acetyltransferase] [protein_id=WP_042231515.1] [location=complement(245754..246260)] [gbkey=CDS]
MNKVQFDEVFELMEASFPDTEIRTCEGQRALLAHPAYRLNVSRNESGEILAFMGVWEFEAFRFVEHIAVNPEQRGGGIGK
KLMAAYTSGSSTPVILEVEPANDTDSKRRIEFYERLGFHLNAFDYFQPPLREGQPELALQIMSYPEPMSEEEFQPYKKVL
MAEVYKQA
>lcl|NZ_CP026520.1_prot_WP_042231519.1_199 [locus_tag=PC41400_RS01050] [protein=zinc ribbon domain-containing protein YjdM] [protein_id=WP_042231519.1] [location=complement(246429..246770)] [gbkey=CDS]
MSNLPNCPACNSEYTYEDGNLLVCPECAHEWNVDAEPGNGEEEKVVKDANGNILKDGDSVSVIKDLKVKGSSSVLKIGTK
VKNIRLVDGDHDIDCKIDGFGAMKLKSEFVKKI
>lcl|NZ_CP026520.1_prot_WP_042231521.1_200 [locus_tag=PC41400_RS01055] [protein=metallophosphoesterase family protein] [protein_id=WP_042231521.1] [location=complement(246915..248288)] [gbkey=CDS]
MMYKFSRSKWLVAPMAVFMAVSVVLPGVDAGVLGSSIAAAAGSKAPTALALTFNGDAKTSQGFAWYTDPAVTGTKLQVVE
SSLVTGGVWPDSGVISFEGSTAPIQVYQSSSDKSAGKKTAYADHKVTATGLKPGTEYTYRAGDGQEGNWSEPGTFKTEEQ
SGGAFTFLYTTDLQGTTEREYATWNHTLEEAVKKFPAARFVTVTGDLVDNGDIENQWFWLLGKPQAILKKTPLVPALGNH
ESKSNNNFWYHFNLPNTSNTGAKPDGSVYAFDYGPARFLVINTEYNEAKGTDAVYRKQEEWLRAEAAKTDKKWKIVLFHK
SPYSVANHSSDSDVLFFRKKLAALFDEIGVDVVLSGHDHTYTRTYPMYNNVPQKDAAVDKDGNLINPKGTLYLVSNAAGN
KRYTPKSGPFPYAYKYGQPGKEMFTGVTVANDKITFQVYTTTESGATALYDQFGIVK
>lcl|NZ_CP026520.1_prot_WP_042231525.1_201 [locus_tag=PC41400_RS01060] [protein=GIY-YIG nuclease family protein] [protein_id=WP_042231525.1] [location=248488..249639] [gbkey=CDS]
MNLAEKVKALPLTPGVYLMKDRLGHIIYVGKAKSLKKRVQTYFQHSKAHSPKVKRLISNIRDFEVRLTDTEFEAFMLECR
LIKQLKPLYNKKMKNPLAYTYLVIRNSREGLRRLDITNDPAGQEGDLFFGPYTSRSTVEKAVQGLQESYRILCSNPYARR
SPCLNHSLGLCLGMCTGSEDALRAYNSVLDKMITSLNGSPPIVLQDLEQRMMEAAESHNFEAAAKYRDLMKTFGSLIRGE
KVIGFAEEDRNIVILEPVNEREAKLILVKGNRIVRDTKVEAPDADRNGPEAPHKLHTLVRSAILGTFRQADPAAPRPKIS
RHDIDEAQILYSFLQGSARSYLIVPEHWLIPGHEGELGEAVEGLIAGYVRAQNSAGDQDGPDA
>lcl|NZ_CP026520.1_prot_WP_042231528.1_202 [locus_tag=PC41400_RS01065] [protein=hypothetical protein] [protein_id=WP_042231528.1] [location=complement(249665..250729)] [gbkey=CDS]
MKNRFDELRRVSWTMPDGKQKLAVLEEMIRIADLYMTEEDSYNVRMDYTSAAMDAGFSEKMLISFAWCVAKFDKNPGRYP
HFYLLWHYKWILNGIWRMPEMTFEQIERMFEDFKERCIRHGHNLRAYHQKRVNLFISLGMFQEAAESYKLWRSTPRDSLA
DCQACEQNLFGEFSFKMNHLKRGMQAVKPILEGRMVCGSVPQNTYSLIIIPLMKLKEYDRAVAIAKKAVRELEGPQYLEE
YGVFLEFFTITDMTRATKLYEQTIRYGLESKIGWSRLQYFYAVRLFLNEWGKTKRRKRLAESDRVTLPWIDHEIAKLTEA
FNRRNGSLYVNEFLTEKQKHTERLVAAYRNSTSQ
>lcl|NZ_CP026520.1_prot_WP_042231530.1_203 [locus_tag=PC41400_RS01070] [protein=HSP90 family protein] [protein_id=WP_042231530.1] [location=complement(250726..252501)] [gbkey=CDS]
MNFQVNLQGVIDLLSNHLYSDPGVFIRELLQNGIDAVTARKRLGQEFEGEVRVEVFSSNTISFSDNGCGLTEKEIEQFLA
KIGSSTKREDQDLADDYIGQFGVGLLSCFVVSEEIVLITRSSNGEGDSLEWRGKPDGTYSIKKLERQVPVGTTIYLKAKT
DYEEYFEFYRLRELLEKYGEFLPTPIILTEDNYELNVNTSIPPWNMDKQEAMDFVEETAYSKPMDIISLTSVIGGVKGIA
SILPYAVSLQDEKKHRVYLKGMLLSDKMSNVLPPWAFFVNGIINTENLRPTASREAFQENDLFFSVRDELGECIKNYLIE
LSQENQELFKRIVLVHYASIKTMAVEDEELYELFIQHLSFETSFGDLKMAEILENHTHLLVAPTLDEFRQVARVAKAQDL
MVINGGYVHDLDLVLNLRNVKDDISVDVLDILEFANRFKELNVMEKQETRPFLDYSNEILRSFQCRSVVRWFEPADIPVL
YNTNQEVNFFKLAEESEKEANPLFAEAVRVIKNELYDQPFARLCYNYNNPIVRKAMESRDLEAQKACIELFYTQALLLGN
HPLSSAELSMMNESLFHFMNMGLDRMPEVEQ
>lcl|NZ_CP026520.1_prot_WP_042231531.1_204 [locus_tag=PC41400_RS01075] [protein=ATP-binding protein] [protein_id=WP_042231531.1] [location=complement(252868..254082)] [gbkey=CDS]
MEYPIRILLVDDRPEDFLTIQALLVDTPYQLIRASSGMEALKFLLESEYALIIMDVLMPDMNGFETAERIKRRKKSEDIP
IIFLTALTPEVEDYMKAYTAGGIDYLTKPFHSGVLRSKIEGFVRLYKARKELQIKTGELQKQTLELEDANRVLTQLKETA
EVASRIKSGFLAMMSHEIRTPLNGIIALSDILMASDLKQDDAEMAEIIHTSGHALLSVINHILDFTKIESGKMELDYEPF
NLTVCLKETMDLFRALARERNLLLETHIEPGIPKILIGDSNRLRQVLNNLIGNAVKFTETGSVKATVSQKRVDDKLITLE
FIVEDTGIGIPADKMHHLFQPFTQIDATISRKFGGTGLGLSICKTLVNLMGGTIYARSNVEKGAVFVFTIQTGIPADTLN
NPSC
>lcl|NZ_CP026520.1_prot_WP_042231534.1_205 [locus_tag=PC41400_RS01080] [protein=response regulator] [protein_id=WP_042231534.1] [location=complement(254125..257538)] [gbkey=CDS]
MNIKGKVWLGTAVSILVIGGLTAMACIFLDGVFLALSLILGLIGLATGVYLITGIHRGINKGMHRVSQIVGDMAQGTVDP
SDRSAEAGDEFGQLAQSFRSLAVDLYNKTEEERILRLEAEEQAWINVHVTEMAILLQGSVQIGTASRIFIGKLASCVQAA
YGALYVPKDGLLQFAAGYAYDDTAAERRQPIALGSGLVGQCALDKQTILLHDLPDGYVKIHSALGETKPAALIIVPVCYE
DEVVAVIELASLRDFTSKERQLIDQTSANMGILLNTLADVVRIGELLKETQLQKEELEAQTEELQSQTEELQAQTEELQM
QAEELESQTEELQAQTEELQAQAGSLQTSNERLQEQMKLTELQKEEIKQQADELKEQAEELFESNKQLQEQMELTELQKT
EIAEQADELKEQAEELLASNENLKQQVELTERQKIEIKEQADEIFMAAQYKSEFLANVSHELRTPLNSLLILSQILAENK
EGNLHSRQLEYIHTIFSAGKDLLTLIDEILDLAKLEVGKMTAVVEPTYLKDVSSHLYRHFEQQAKEKNIHFDIHSDRRLP
ESILTDGHRLQQILNNLLSNALKFTEKGAVTLSVHGASAPTGGDGQAPAEEIVFSVTDTGIGIPASKLESIFEAFQQADG
TTSRKYGGTGLGLTICRELAGLLGGRIEVSSAEGEGSTFSLHLPAAQPGIAVSQAAAASDTMSAASEQAKRGESSFQESF
VPDISISNPKLLQLAEMEDDRDNLQPGDTLLLIIEEDAEFASTLLNLARQRNFKAIVASQGDQGLALAHAYKPDAILLDI
QLPVLDGWSIISRLKGRPELRHIPIHVISTAEENQQGLAMGALAFWKKPNDVHELEAAFLQIESYIRRRTKSLLIVEDNP
VLRSSVVAFIAHPDVNIVAVATGREALEQLASHHFDCMVLDLGLSDISGFDLLEQVKTNRKLQTLPVIIFTGKELNKQDE
QRLHHYAESIVIKNVRSMERLYDETALYLHRKHADLPADKQQLIEKLHNPEAAFQDKNILLVDDDMRNIFALSSVLEGYN
MNIQFAQNGKEALSILRNHPEVDLVFMDIMMPEMDGYETMREIRKMPEFEKLVIIALTARAMDEDRIKCLQAGASDYISK
PINTTQLVSMLKTWLIT
>lcl|NZ_CP026520.1_prot_WP_042231537.1_206 [locus_tag=PC41400_RS01085] [protein=S26 family signal peptidase] [protein_id=WP_042231537.1] [location=complement(257720..258310)] [gbkey=CDS]
MKKWLLAALLLGSLAACSEKAIDDKVTKESVAEVSAGNHTVVNPLSDAMNRGRHELEKNVAVDTSYYKSRDVARGDIVYY
GDHDVSRVVALPNEKINITNGQIYINGAKLTAFYGAAHIRGYDYKAFQNSDIEEPAKQNLIEEIYKKEVQEITLNGDEVF
VIGDDWSRSNTSTFRSLSLNELKGKVVGVCTTCSTP
>lcl|NZ_CP026520.1_prot_WP_042231538.1_207 [locus_tag=PC41400_RS01095] [protein=sigma-70 family RNA polymerase sigma factor] [protein_id=WP_042231538.1] [location=258701..259288] [gbkey=CDS]
MNDEQLEHWLDRLAAGDQEAMEEIYRMTRRKVYGTVAVLVSNKEDVNDIVSEIYCQLWRALPGYDRSRPFHFWLNGIVIR
QVSSWRRQLWRRFRLTDKTSRLSEEPHVETPDEALAGNELRQDVRDAINKLPYKLRVAVIYRYFYEYTYDEIAELLAIPP
GTVKSRVHLALKRLRAKFDTQLDGEAYPNVYSKKH
>lcl|NZ_CP026520.1_prot_WP_042231540.1_208 [locus_tag=PC41400_RS01100] [protein=DUF4179 domain-containing protein] [protein_id=WP_042231540.1] [location=259266..260048] [gbkey=CDS]
MSIPKNIKEALKDQIENAEVPQEVDRRVRQSFLEFHKKKESKPMKKRLVAFSIAAAILIPTSVFASLGGTSYFFKESANI
NGLVNEGTKRALSEGLSVPMNEKITDHGLTVQFSEVYVEDSKVLVHYKITGEDGKLVPYEFDTRGLNVLYDGKKNGKQVE
NPTYQENGLDGFSVLSFIGTGKKDRLPFTLTDAAGNVINTGIAEKDQPEGVLAFVTSGAKLPESIRLNVNIDRIGATKGS
WKAQFTIDQSKAKQASEKAH
>lcl|NZ_CP026520.1_prot_WP_042231542.1_209 [locus_tag=PC41400_RS01105] [protein=glycosyltransferase family 2 protein] [protein_id=WP_042231542.1] [location=260282..261022] [gbkey=CDS]
MKDGSIKSVKNLGVSVITCTNRRNYLKNLLDNYKRQDHPTKELIIIVNNDKIPLAPYRELAKKLRNIRIFRVSERTSLGA
CLNYAVKKAKYGYVAKFDDDDYYAPYYLTESLLTFRQTKADVIGKRAHYMYLRGSKTLILRFSQDENRPVTQLPGATLVI
RRDVLKKVPFPDRNVGEDDLFCLRSKKKGFKVYSAGKDNFVAIRRKNSSNHTWIISDRELIAHHKKVSNVNDYKKFVQRK
PKSGVL
>lcl|NZ_CP026520.1_prot_WP_042231544.1_210 [locus_tag=PC41400_RS01110] [protein=glycosyltransferase] [protein_id=WP_042231544.1] [location=261022..261753] [gbkey=CDS]
MAKPDVVRPEHAVSVITCTKRRDCMENLFQNYSRQNFRHKELIVILNHDNLNVSEYIKAAKAYKQVRIYSRPGLVSLGNC
LNFGVRLSKYDRIAKFDDDDYYAPDYLTDSMRTMVRTGADIVGKRAHYMYLNGTKTLLLRYPARENQYVSLVQGATLLVK
RHVFDKVGFPDRNRGECVKFCADSAAKGLKIYSCSRYNFLAIRRKDSKDHTWTVSDRQLLARNAKLVKADNPQKFVSRRS
LDS
>lcl|NZ_CP026520.1_prot_WP_042231546.1_211 [locus_tag=PC41400_RS01115] [protein=YdiU family protein] [protein_id=WP_042231546.1] [location=complement(261914..263395)] [gbkey=CDS]
MTDKKAGPETGWNFDNSYARLPEIFFAKTNPTPAASPDVVIFNDALAASLGLNSQALKSNEGTEVFAGNRIPEGAEPLAQ
AYAGHQFGHFTMLGDGRAVLLGEQITPGGERVDVQLKGPGRTPYSRRGDGRAALGPMLREYIISEAMHGLGIPTTRSLAV
VTTGEPVYRETELPGAVLTRVAASHLRVGTFQFAAKWGSAEDLQALADYTLQRHYPEAAAAGSGNRYLSLLQAVIKRQAA
LIAKWQLVGFIHGVMNTDNMALSGETIDYGPCAFMDAYDPETVFSSIDTYGRYAYGNQPHIGAWNLARFAETLLPLIHAD
QEQAIKLAEEAIAQFGVLHNRSYLAGMRAKLGLFNEEEGDESLIEDLLGWMQKHRADYTNTFLALTFDKPEDTALSGAEE
FTRWHELWQARRGRQPESAAASHELMRSSNPAVIPRNHRVEEALEAAVEQGDYSVMERLLAVLASPFAHSPEQAVYSTLP
EACDSPYRTFCGT
>lcl|NZ_CP026520.1_prot_WP_042231548.1_212 [locus_tag=PC41400_RS01120] [protein=ABC transporter permease] [protein_id=WP_042231548.1] [location=complement(263749..265668)] [gbkey=CDS]
MTFRQFAFNNVIRNKRVYLAHFLSSSFSVFVFFVYALLLFHPDLQGELRSTSRTISALATDGMRLSQVILFIFSFFFCLY
SAGTFLQTRKREFGILMMHGMSPRQLRRLVFIENLFIGGASIAAGIAVGLIFTKMILLGIAKLLVIKVGLTFYIPVLAIV
TTAAAFALLFGVVALVTSRIVKAGRLADLFRSDEKPKPEPKPSRLLSVLAIVLILAGYGVVMVYVYRYFHSFWLLFAGVG
LVVAGTYFLFTQFSVYALRSLKRKDRVFFKRTNLLVLSQLAYRMKDNATMFFMITIVSAIAFTGIGTVLATNHPGLKEME
NPYAFTYKMDLSVQKETGVVSLIKKQLADHGFAYDLNVVHAKRLWGRLILVKQSEFNALAKTLGLRDTEELAEGESLLLP
GSLFLATQIKEKKEIVPDSIDLGISETWQKNVTVKQYVPTVILPDGHYTVVVSDRMFETVPEIENTQLQFNFFYVADWQN
TRDVASALAEEIGKNEFDRFAFDALVLDWLSDKQTNGLLFIISAMVGVVFFTFAASLLYFRLYADMERDREQYRMIAKMG
LLNKELRRIVTRQLLLMFFLPIVLALVHSGVAFIALQHLVDFSVWMNSVLIFLSFIGIQVVYFFLIRWRYLQHMYPKNG
>lcl|NZ_CP026520.1_prot_WP_042231551.1_213 [locus_tag=PC41400_RS01125] [protein=ABC transporter ATP-binding protein] [protein_id=WP_042231551.1] [location=complement(265643..266413)] [gbkey=CDS]
MDILKVTNLCKIYNEKMPFEALSDINLTIEKGEFIGIMGPSGSGKTTLLNLVSTIDEPTSGEVCVDGNHPHRMNADELAL
FRRRKLGFIFQSFNLLPTLTVKENILLPLTLEGVALDQMKILLEPIAEQLGITPILDKRTYEISGGQAQRTAIARALIHS
PKLVLADEPTGNLDSKSARDVMELLTAMNKRQEATMMLVTHDAMAASYCDRVVFIKDGRFHNEITQGGNREVFYQKIINA
LSLLGGYRHDFPTVRV
>lcl|NZ_CP026520.1_prot_WP_042231552.1_214 [locus_tag=PC41400_RS01130] [protein=sensor histidine kinase] [protein_id=WP_042231552.1] [location=complement(266568..267551)] [gbkey=CDS]
MRLFVRDHVPLIGFSLVQMLVILLICWLDGYNHLGTALYALFLGLCVLLVYLMYRYVRLRAFYRYISDPSDVLPRELKVS
RAPLAASLTKILDARYRESKQLEHVREERQRQHLTFMNQWVHQMKTPLSVIQLITQNGEDERFASIAEESDRLRHGLEMV
LYMARRETFAQDFHIYRVGLKELVNEVIQDNRRSFIRHYVHPELLIDESVTVESDAKWLRFIVQQLISNAITYSAGSRTK
VVFKAHAQDKAVILSVTDQGVGIPPSDLKRVFDPFFTGENGRKFKESTGMGLYLVKEVADRLHHRVEIESETGKGTAVRI
IFPYATE
>lcl|NZ_CP026520.1_prot_WP_042231553.1_215 [locus_tag=PC41400_RS01135] [protein=response regulator transcription factor] [protein_id=WP_042231553.1] [location=complement(267548..268249)] [gbkey=CDS]
MNKIMIVEDDPKIAELLKSHIQKYGDEAIVADDFQRVMELFQEQLPQVVLLDVNLPSYDGYYWCRQIRTVSTCPIIFISA
RSGEMDQVMALESGADDYITKPFHYEIVMAKIRSQIRRVYGDYATKERIVERDGLVLYPERMELRLKEQSTLLTRKESLL
METLMHHCPRIASRETILDKLWDDYSYVDDNALSVNITRLRKKLAELGIDDAIETLRGVGYRLHLTWKGETKR
>lcl|NZ_CP026520.1_prot_WP_241688769.1_216 [locus_tag=PC41400_RS01140] [protein=MerR family transcriptional regulator] [protein_id=WP_241688769.1] [location=268477..269487] [gbkey=CDS]
MSGLYSKSGTDSQKQQASDSIDQERTYSIGDVAKIIGSTIKTVRYYDEVGLVKPTTYTEGGHRLYTTQDIWRLELTITLR
YLDFGIDEISQIISRETPVDKALDWQIESLELQVSTLSNMISILRMAKENEGDSLQYVYDLVHERAVNNEKRSRFIAKRV
EESKFLEGVPAEWRSSLLYNFNKHIVNQTKTSAKQTVAWNELQELINDPEFISDLRKDEFLFFNMVHQPRLGADAWVKNL
EGIHTRLNTALKRKLGAESPFVQAIVEDTAMLYANSEQSVDQEEFFRYFAEYSQRTPSERIERFNTLCSLLSPKFRLFSQ
GNLLLLQGIKWRLEQM
>lcl|NZ_CP026520.1_prot_WP_241688770.1_217 [locus_tag=PC41400_RS01145] [protein=acetylhydrolase] [protein_id=WP_241688770.1] [location=complement(269553..270944)] [gbkey=CDS]
MAKNKSRRGLLIGFGVSGLLVLTHGLTEGMRWQMIPVYAMTLIPIAIFAVRYRKPKEEEKKAARTRVILITALAALYSSA
AVALPVLLPVFTFEKPAGPYKIGTVSYHWKDDRREETHTPDAGDKRELMVQIWYPANMNAKGKTASYVSHGDIFADGYGK
ALHLPKLLFTSIGMVKTHAIEKAELSDKEPAYPVLVFSHSLNGTKNQNTFEVEQLASQGYIVVGIDHTYRSTTSVFPDGR
VVNFVPQEGNTVAYLDKINGEWVEDAKFVVDQIERLAKNDPDARFTNRMDLENMGMFGHSFGGATAVQMLMADSRIKAAI
NMDGGLYGELRIPAEGLGKPFLMFSADDTLSGTKNMSDENIASQGTTREDLTKFFDETLARYKDITVGGNYWIKMKAMKH
MGFSDMYLISPVFEKMEGVEVRKAHRLINEYSLDFFNHYLKQKPFNLLEQTIGEHQGFTLEKG
>lcl|NZ_CP026520.1_prot_WP_042231556.1_218 [locus_tag=PC41400_RS01150] [protein=pyridoxamine 5'-phosphate oxidase family protein] [protein_id=WP_042231556.1] [location=complement(271217..271705)] [gbkey=CDS]
MSTYEKNQEAIEKVRDLIKDIDVAMLTTISSDGLVSRPMSTQEVEFDGDLWFMTQKDKSVYRELLENKNVNLAYVDKSYV
SVRGEAELIEDKAKLKELWSPMYEKFFGVSSDDPSLVLIKVKAETAEYWEAGNRMKLVKNMFKKLTGKENDESDLNKTVE
LN
>lcl|NZ_CP026520.1_prot_WP_042231557.1_219 [locus_tag=PC41400_RS01155] [protein=alpha/beta hydrolase] [protein_id=WP_042231557.1] [location=complement(271851..272789)] [gbkey=CDS]
MDFENRVLPELRPIIAEFPGFQLEENLELSRSYLSNPPIQKSEHVRTTSRMIPSLAGEMLVKIYEPSQRTGAKLPAMLWI
HGGGYVMGHPDMDDALCERFVQAAECVVVSVDYRLAPEHPYPAAIDDCYTALTWMTDEAELLGIDLKRVAIAGASGGGGL
TAALALMVRDKGGPSLIFQMPLYPMIDDRNITASSFEITADNATWNRANNLAAWSMYLGNRIEDSGVSPYAVPSRAESLA
GLPPTYTCIGQLDLFRDETIEYVTRLAQAGVDVEFHLYPGTFHSFEVMVPQAEVSRRASENYVDAIARALNP
>lcl|NZ_CP026520.1_prot_WP_042231559.1_220 [locus_tag=PC41400_RS01160] [protein=LLM class flavin-dependent oxidoreductase] [protein_id=WP_042231559.1] [location=complement(272779..273873)] [gbkey=CDS]
MEIGITSFVETKPDIRTGEVMSHAQRLREVVEEIVLADQVGLDVFGIGEHHRQDYAASSPALVLSAAASQTKRIRLTSAV
TVLSSVDPVRVFQDFATLDGISNGRAEIIAGRGSFIESFPLFGYDLNDYDELFEEHLALLLKMRESERVTWKGKHRPTIN
NLGVYPRPVQNSLPVWIGSGGRQESAIRAGLLGLPLMLAIIGGRPTDFAPLVQLYKKAAAHAGHDESQLRVGSHSIGFVG
ESTEHAADTFFPSTMAGMNRLGKERGWAHYDRASYDAARSFEGALYVGDPETVAQKIIHLRKHVGVTRFMLYVPLSTMPH
AEVMRAIELLGTEVAPRVREEITKWEAETEKRSTFNYRSDSNGF
>lcl|NZ_CP026520.1_prot_WP_042231872.1_221 [locus_tag=PC41400_RS01165] [protein=MFS transporter] [protein_id=WP_042231872.1] [location=complement(274029..275198)] [gbkey=CDS]
MPRYATLLFAAACGLSAANIYFAQPVLDKLSSEFSIDYSIIGILITITQIFYAAGLLLLVPLGDLLNQRRLIIGQMLVSI
AALLIVGTASSGTALFAGLALVGLLAVVTQTIVAFAATMACPAEQGRVVGVITSGIVIGILLARTVSGILTDIAGWRSIY
LVSAALMFLMVCALFRVMPNVERRVQSLSYLQLLRSVFMLFAQERIVRIRAILALLIFADFSILWTSLVLPLSVPPLSLS
HTAIGAFGLAGVAGALAAAHAGRLADRGYGQRTTGTALTLLLVSWLLICYAERSLFALAIGIILLDLAVQAVHVTNQSMI
LSLRTEARSRLTAGYMVFYSIGGAIGSVSSTHIYARYGWDGVCLLGASVSALALLFWAMTRRAAAAALK
>lcl|NZ_CP026520.1_prot_WP_042231561.1_222 [locus_tag=PC41400_RS01170] [protein=TetR/AcrR family transcriptional regulator] [protein_id=WP_042231561.1] [location=275567..276148] [gbkey=CDS]
MGRIREFDEDQVLDAAMQLFWEKGYEATSLSDLTSRMGIQRPSIYSAFGGKKELFEAALRKYTMTRASLIRAKLQSNPSV
KEGFRTFFEDVIVEEYAENPRKGCFCINTMVELAPHDEKFEILTREHQMYLAVIFEETIERGIQSGELEIGVNAKALSQA
LLLSLIGLTVLIKSRPERSFIDNSIEVILTLLK
>lcl|NZ_CP026520.1_prot_WP_042231566.1_223 [locus_tag=PC41400_RS01175] [protein=SRPBCC family protein] [protein_id=WP_042231566.1] [location=complement(276291..276731)] [gbkey=CDS]
MAIGENELVTTREYDVPRELVFRAWTTPELLARWWGPKGFTNTFHECDIRPGGTWILTMHGPDGTDYPNRSVFVEVVAPE
RVVLDHLSGHEFRLTGTFEDLDGRTKVTFRQRFKVKEEFEAAKPYCAEANEQNLDRLGEVLAELAG
>lcl|NZ_CP026520.1_prot_WP_042231567.1_224 [locus_tag=PC41400_RS01180] [protein=SRPBCC family protein] [protein_id=WP_042231567.1] [location=complement(276734..277177)] [gbkey=CDS]
MSERFVKHATFVVERVYAASPKRVYHAWADPDAKAKWFSKPDIFEFRVGGREYSSGGPPEGPVFTFDASYQELVPDQRIV
YTYTLDSDGTRISVSNTTVELIAVEGGTKLIFTEQGAFFDGHDTPEVREHGTNAMLDGLGKALEGEE
>lcl|NZ_CP026520.1_prot_WP_042231569.1_225 [locus_tag=PC41400_RS01185] [protein=MerR family transcriptional regulator] [protein_id=WP_042231569.1] [location=complement(277174..277680)] [gbkey=CDS]
MKRHWKVGDLAKLTGLTVRTLRYYDQIGLFSPSAQTESGHRLYSESDLSRLHQILSLKELGLSLEEIGSVLTGGQISPLE
IVHLQIDRIKEQIRRQQKLLEQLGHVSKLMQGKTQVTVEDFTDLLQAMQMGFEKPVIERQTSWEQHLDRLGDFLAGGSGI
PKPEEDKQ
>lcl|NZ_CP026520.1_prot_WP_042231572.1_226 [locus_tag=PC41400_RS01190] [protein=hypothetical protein] [protein_id=WP_042231572.1] [location=277884..278213] [gbkey=CDS]
MVSSRIEPKGTANSRSDYRHQGNWALECGFWHNGEKALCKARIFLKPYPASSIENCPVQPFSTKKTGRGNVRLFKEHGTF
WVLFNRNLIKDSLFFFFVKRRISGSAMYS
>lcl|NZ_CP026520.1_prot_WP_042231574.1_227 [locus_tag=PC41400_RS01195] [protein=MarR family transcriptional regulator] [protein_id=WP_042231574.1] [location=complement(278210..278623)] [gbkey=CDS]
MEEQTVFELIHNMDKFTNGLIIQWNKTFNEDLGVSHVLVLGHLKQNGKSRPSDLAKILGLTPPTLSYLSEKLVAKKLAVR
MVDESDRRIIYLGITDKGAEVLKRATLEGERLRRNLFEKLTDEERAELANLFKKLNS
>lcl|NZ_CP026520.1_prot_WP_042231576.1_228 [locus_tag=PC41400_RS01200] [protein=SDR family oxidoreductase] [protein_id=WP_042231576.1] [location=complement(278692..279426)] [gbkey=CDS]
MGQLDNKVAIITGGASGIGERMVDLFAQEGAIVIAADINEEALARASKKQNVHGMKLNVASDEDWAALAKEVNDRFGKID
ILVNNAGISSEKPFQEIDAQDWQKMLSINGFGPFAGMKHVTPYMAEQKQGSIVNISSYTALIGQGFNHYSASKGAVRALS
KAAAAAFGRQGIRVNALFPGIIETPMTQALSTSKDLVDRLIQMTPLQRLGQPDDIAKAALFLASDASSYITGSELVIDGG
YSAQ
>lcl|NZ_CP026520.1_prot_WP_042231577.1_229 [locus_tag=PC41400_RS01205] [protein=SDR family oxidoreductase] [protein_id=WP_042231577.1] [location=complement(279440..280189)] [gbkey=CDS]
MKLQGKVAIITGSASGIGRGIALAMAKEGAHIAIVDVNEQKGQETLAEVNQYTEGCLFIRDISVKENANEIVEEVVKKFG
KLDILVNNAHASRQASFVETTQELLDLSFGTGFYPTFHFMQAAYPELKKTKGKVINFASGAGLDGQVTQTSYAAAKEAIR
AISRVAANEWGPDGINVNLISPIALTPGVEQWRESAPHLYDAMIERIPLRRLGDPEQDIGRVALFLASDDSNYITGQTLM
VDGGSIKLR
>lcl|NZ_CP026520.1_prot_WP_042231580.1_230 [locus_tag=PC41400_RS01210] [protein=MFS transporter] [protein_id=WP_042231580.1] [location=complement(280233..281501)] [gbkey=CDS]
MSQNKGKIHYAWWILLGLCMMVGLGKGTLNNSAGLFLRQVSKDLGVGMGSLTLYFSVSAIVTMVFLPIGGKLLAKYDTRL
LLSGAIILQAGAFALFGLMNSVWGWYILAVPLAVGGVFITVIAGPVLINQWFKKSKGLALGIMGAAGGAIGAISQPVVGN
LIANQGWRSSYVIVGIAVIVIVVPIVLLLLRKSPLEKKTLPYGAEAAGGTAANATAAGAENKGVTLAAAKKSSAFYALIA
FFFLITSAASFSVHIPTYLMNKGFDVTFAGNVMATNMIGVLIGSLVIGYASDKIGTRNTTFIAMILGVLAIGLLLFAATS
TLVISLAVGLFGLVSASIGTLGPALTSSLFGNKEYSQIYSNASMGLAISSIIALPAYGFVFDFTGSYTPVLYAIGIMLII
NILCISIAFGGKKKLEQAGLWN
>lcl|NZ_CP026520.1_prot_WP_042231582.1_231 [locus_tag=PC41400_RS01215] [protein=TerD family protein] [protein_id=WP_042231582.1] [location=complement(281831..283933)] [gbkey=CDS]
MNINTIYLRRAGKVVVEQGKGAGRLSKAYLAAAMKNIESLGFTFSHRLMRVVCTLSKERFEAFYRELIGDLKRMVGAHVA
YKPMYPDFPAQVMQEQDAELYLNAFYHYLTHELPEYETAQRLPLQDDADLKVIDLGNNAEFHVLISRLIQAKTSISGTDK
EDIDAAIECADPDELDAILPSEIPFKENAAFVAASLMKHDKAGLERVGRYFKTATDVLRLAAAWSDGDVSLAEAVRYRKF
KRSERRLLLGLLERCHPITEDMLRYKERWVRLGEILHPSEYKNRFPRCQEAFGILRDNKPFTTFGGSVELAFRYGQVWTL
IDLLMQRPGEFARRLDQLLRSTEQTEYVVLAFGEVAGQVSTPVLLQVKNHFARRHERPELRVFFPKGNAAKAFAIPHTLP
EISETACQDIVELCEKTLIQRFSELPPLGKTHVDKRLKDYHVPFSQRSASKALHTLVRGSRVPMAEGDTIRFFNWWKEGT
VDGKPTGRVDIDLSAVMYDRNWEYVEHISYTNLRSSKYQAVHSGDIVSAPQGACEFIDLHIPSIVDYGGRYIVASLHSFT
EQPYCDLPECFAGWMMRKKPGSGEIFEPSTVVNKIDITADTRIAVPVILDLVERTVIWTDLSLTKQPHYCNNVEGNQKGI
VLIGKAMTTLRKPDLYDLFLLHAKARGGAVDTADQADTIFSVDSGVTPYHIEQIMAEYVV
>lcl|NZ_CP026520.1_prot_WP_042231585.1_232 [locus_tag=PC41400_RS01220] [protein=ankyrin repeat domain-containing protein] [protein_id=WP_042231585.1] [location=complement(284290..285375)] [gbkey=CDS]
MIQLKDIGKFEKLPEIAMHIYQGNMPALREAIAAGWDIEEGIELSKYTTLSPLDLALISERLDIVKLLAENGVNLNVPRN
PAFLRAIRYCKEDIVRYLAGQGAKADKLNHVGSGAYSQAYYGNKNNIPLLHELGLDIKKHGSPVLRQAVSDHDLKTLGYL
LDHGADINYNKPDQVYPYQATPLTVAVRIGNVAMVKYLIERGADITITEKDGDRAYTIAVGNKNAALADYLKSLEPAELH
DRENKKFELKKYKLTDELVAFLTGDKLRLELAPNDYDIGYIDFFTLTDTIEMKVGRQKLLRLSADIDNYSDLQLVWNPKK
KGTVGCYDVEHQEYADLCGFTEFLAHPETYLIKFLEGELQE
>lcl|NZ_CP026520.1_prot_WP_255246854.1_233 [locus_tag=PC41400_RS30515] [protein=hypothetical protein] [protein_id=WP_255246854.1] [location=285567..285698] [gbkey=CDS]
MSAYDVDIGKLISEKRELEGGGSYLLCTPDKTIIEVDSLGPND
>lcl|NZ_CP026520.1_prot_WP_042231587.1_234 [locus_tag=PC41400_RS01225] [protein=hypothetical protein] [protein_id=WP_042231587.1] [location=285720..286520] [gbkey=CDS]
MYTFSIYTDTAHGSFVIYINNLESLNRSVEKALDRDRQRYPDKWDANGNTAREQLYFEFKYAEADYPFMYDSMPERLQKW
LRIYACVSMEQPHHLEIEQNYIFEKQLFDSQFFLIAIDVIHRLRNDFQQLQRTGTFIAYVSAADGVGGDPLTTSQLLRRC
VNEEQLYKAMPEVQEQDEAFRMAVSAVRQHSLREQVLHWVTVIEEGEFSEDSPYSFWKTDYEAYEQLLELGGPAVPFIRE
HLNGELKLDTRLILEMVMRDLGEIVP
>lcl|NZ_CP026520.1_prot_WP_042231589.1_235 [locus_tag=PC41400_RS01230] [protein=SDR family oxidoreductase] [protein_id=WP_042231589.1] [location=complement(286662..287426)] [gbkey=CDS]
MRKLEGKVALVTGASRGIGRGIALRLAREGAFVAVHYGQRRNEAEAVVRQIGQSGGNACAIGADLSTLGGIEHLFTALDE
AIREHTGGSGFDILVNNAGIGQILSLEETTEASFGEVMKINVEAPLFVTQQALPRMKDGGRIINVSSFVTRVASPSVFTY
SMSKGAIDTFTLLLAKQLGSRKITVNAIQPGIINTEMNAGTLQDPNGQKNAAGLSVFNRWGEPDDIADVAAFLASPDSRW
VTGQLIDASGGSHL
>lcl|NZ_CP026520.1_prot_WP_042231591.1_236 [locus_tag=PC41400_RS01235] [protein=TetR/AcrR family transcriptional regulator] [protein_id=WP_042231591.1] [location=complement(287574..288155)] [gbkey=CDS]
MNGKRGRPRNVEAQKSILSASYELLLENGFQAVTVDKIADRAQVSKATIYKWWPNKAAVVMDGFLYAATARLPVPDTGSA
CKDILIHATNLTRFLTSREGTIITELLGEGQFDSGLAEAYRTRFFRPRRLEARGLLEKGIQLGELKNGIDIDICTDLIYG
PIFYRLLVTGDKLDEPYVQYLVLTALEGIRSKA
>lcl|NZ_CP026520.1_prot_WP_042231593.1_237 [locus_tag=PC41400_RS01240] [protein=MFS transporter] [protein_id=WP_042231593.1] [location=288436..289647] [gbkey=CDS]
MNALKKQTTEKNIPAWLTLLLAAACGIIAANLYYAQPLVGEISTSIGLSANSSGFIVTLTQIGYVVGLLFVVPLGDIVEN
RRLVVMSLLLTGAALAVTAVSKQAVPFLAASFVIGLGSVAAQVLVPFASYLASETSRGRVVGNVMSGLLLGIMLSRPLSS
LAADFFGWHAVFALSALAVIILAVVLSKALPVRKPSTDTSYTALLGSMWHLLRTTPILRRRAAYHACVFATFSLFWTTAP
LLLSGPLFHFSQKAVALFALAAVAGAIAAPVAGRLADRGWTKPATGIALAAVAISMLLPLIIRTGSPVGVAVLVVSAVLL
DAGVSANLVLGQRALFSLSPEIRSRLNGLFMAIFFFGGAVGSAIGGWIYATGGWSAALWIGIAFPIAALLYFATERRYNA
QRG
>lcl|NZ_CP026520.1_prot_WP_042231595.1_238 [locus_tag=PC41400_RS01245] [protein=low temperature requirement protein A] [protein_id=WP_042231595.1] [location=complement(289737..290825)] [gbkey=CDS]
MVEKKVTWLELFYDLLFVAAVSKATHVLLHADHGVIPIEYLVKFVLIFIPIWWGWVGQTLFVNRFGQDIFIHRILSILQL
FFILIMTASLSPDFDQYYIPFFIGYIGSRALTAIQYLLVHKSKAAHQQSTASYLGTHFLIGILISSCSLFFDSWVRYAVL
YLGIAVDIILPLIGRKHLVKTPIHTPHLFERFALFTLILLGESVVSILAVLQFSSWDLSSVLFVAFTSVLVIAIWWQYFE
NVETKVSKEIQTAGQTIIYGHLFIYLSMSMIAAAIQLLYQEQLNYTFMTVFVFGSVLLYFFSTSLVFHLYRPKHLRLKPY
HLAVLLGILGCCIALNLIYHVPNYAIVGEMMVFFIAYAKLTT
>lcl|NZ_CP026520.1_prot_WP_042231597.1_239 [locus_tag=PC41400_RS01250] [protein=hypothetical protein] [protein_id=WP_042231597.1] [location=complement(291125..291676)] [gbkey=CDS]
MELSFYLDTHGTCVSELRDELLPLEEAAKTWEFPYAIVVVFRVLPESYERKTFRRFDSRDNLLGLDISLIYEQYLRMSKN
EQRETLGIYLYDYLSESVAKYKKHADKDTQNRFLAQVKSWMLENNWLHGKINKARELLAQDMGLYEVSQELKMPLEEIEY
ILLRMNDYEPADAHPDNRNILIK
>lcl|NZ_CP026520.1_prot_WP_042231598.1_240 [locus_tag=PC41400_RS01255] [protein=hypothetical protein] [protein_id=WP_042231598.1] [location=complement(291740..292480)] [gbkey=CDS]
MYLVSRPKPLSDSTNQALAKDEAAFPPGYLRFLRSFGEGTYRGWMNVQLPDPDVLKPFAEYGLWEHDADSPITERQIAEC
VAISTTVDGDFLAVHPQTDQLLWLPRHGEQVTAIPLQAREREDEATYALVLDEIYRQVYGCGREGAVYYEPWTGSRSHFF
LRLPPGQEQFSLADLAGMCRAAFSPDLSVETEYSCFLFYQALGGYVRFNYANGQEVAVMYEQDEIQAVDVIKQWLLSKGC
EPYPEK
>lcl|NZ_CP026520.1_prot_WP_042231603.1_241 [locus_tag=PC41400_RS01260] [protein=malate:quinone oxidoreductase] [protein_id=WP_042231603.1] [location=complement(292713..294200)] [gbkey=CDS]
MKTDVILIGAGVMSATLGSLLKELVPEWEIKVFEKLAKAGEESSNEWNNAGTGHSALCELNYTSEKPDGSVDISKAIQIN
EQFQLSRQFWSYLVNSKLIRNPQDFIMPIPHMSMVQGQENVTFLKKRFEAMSNNPLFQGMEFSDDPEKLKEWIPLIMEGR
TSSEPIAATKIDSGTDVNFGALTRMLFDYLQTQNVEMHYKHSVDNIKRAADGSWELKVRNLDNGSVERHTAKFVFIGGGG
GSLHLLQKSGIPEGKHIGGFPVSGLFMVCNNPEVIEQHHAKVYGKAKVGAPPMSVPHLDTRYIDNKKSLLFGPFAGFSPK
FLKTGSMFDLITSVKPDNVLTMLAAGVKEMALTKYLIQQVLLSNEKRVEELREFIPNAKGEDWDIVVAGQRVQVIKDTEA
GGKGTLQFGTEVITAADGSIAALLGASPGASTAVHVMLEILARCFPQHIKEWEPKIKEMIPSYGVSLLNNPELFQEILTS
TAQTLGLVEKELAYN
>lcl|NZ_CP026520.1_prot_WP_042231605.1_242 [locus_tag=PC41400_RS01265] [protein=hypothetical protein] [protein_id=WP_042231605.1] [location=294617..294991] [gbkey=CDS]
MTALIVILVIFSLITASVATTIVVLMLRTRSKQRLQSKTILTEGTSAQAVINSIRQTNVQVDDQPEVLLDLTVTQQDGKV
VHALVKAVIQIVNIPAFQKGSVIEVKYMTIDNEQKFAVVGSYMP
>lcl|NZ_CP026520.1_prot_WP_042231607.1_243 [locus_tag=PC41400_RS01270] [protein=DUF4259 domain-containing protein] [protein_id=WP_042231607.1] [location=295097..295519] [gbkey=CDS]
MGSWGTASLESDEGLDVLHALTEYTTERKSVKLKELITHFREIGFLSEDPDEEDFLYDNTAIALAEIVCAYIENGKTQYE
QLNGLNEIVLETEDLRELKQLVQQVLDNKGGDRELYELRDGDSDWIKHLENVVRILSERL
>lcl|NZ_CP026520.1_prot_WP_042231608.1_244 [locus_tag=PC41400_RS01275] [protein=hypothetical protein] [protein_id=WP_042231608.1] [location=complement(295625..296242)] [gbkey=CDS]
MDRLKHSGFYKLKFFITPEEFESVLKLFEPKHAQFYLTNHARTKHDQHEVVEAYQTFYRYFAAQEKPNYYPFFVYSISFS
AGNESSGFFVKNEGIRYPYHGQWAEDELPYVMLSLPKGLQINLEDEKGKYYVYEDIREHQPLTYAFFDELASYTKKITKP
LRFSAYAADALQEQKPPVRISQNAANELADSWIFKKYELVMKGKQ
>lcl|NZ_CP026520.1_prot_WP_053228750.1_245 [locus_tag=PC41400_RS01280] [protein=glycosyltransferase family 2 protein] [protein_id=WP_053228750.1] [location=complement(296430..297200)] [gbkey=CDS]
MEPIELSVVIPVYNSQDTIGAVVEPILHLYNALFSLEIILVNDGSTDASRQECENLSNRYANVVVIHQETNGGQQLALMT
GLRRCQGNLVVLMDDDMQNPPSEIIKLIQKVNEGYDVVIGKRMIYNQSLIRKLVSRLNQFLVFVSTKQKISFTNFLIMRR
AIVTMIIKDQSPKPVIQGLILRSTANLANTLTEHHQRKNGKSNYNLIKLFKHWLTIVRYYMPPFVKYFLFMLIIIVALGA
ATGVYFIIDHFRRNWS
>lcl|NZ_CP026520.1_prot_WP_042231611.1_246 [locus_tag=PC41400_RS01285] [protein=hypothetical protein] [protein_id=WP_042231611.1] [location=complement(297223..297960)] [gbkey=CDS]
MKKIDAYWEKRNFDKKVLEIELDANDSTDCLAELQNDYADYEYIVTKVPKRRMDLVHGLEEAGFRFMETQMEMTMNLTKL
SKPSRFTKAISDHIQFQTVETIERLEEMLSRIDEDLFVTDRVSLDPELGIGLAHKRYVNWIRDGFLSGDALIIEAILKTK
KIGFFYFTKRNDKAIHAVLVSVYNNSRRRGVGLSFLEEIRSWLAEAGYTKAVTRVSSNNIEALRANLFVGFEIKEIYYIL
RKVTN
>lcl|NZ_CP026520.1_prot_WP_042231614.1_247 [gene=rffA] [locus_tag=PC41400_RS01290] [protein=dTDP-4-amino-4,6-dideoxygalactose transaminase] [protein_id=WP_042231614.1] [location=complement(297990..299132)] [gbkey=CDS]
MIPFNKPLLTGQEDAYIREAIESGRFAGDGPLTGRCSRWLEDTLGCDRALLTTSCTHALEMAAMLLDITEGDEIIVPSFT
FVSTTNAFVLRGAKLVFVDIRPDTLNMNEQLIEKAITNKTKAIVPVHYAGVSCEMDTIMKLANQYNLVVIEDAAHALMSR
YRDKYLGTIGHLGCFSFHESKNYQCGEGGALIINDKRFIERAEVIREKGTNRTQYLQGEIDKYTWVDIGSSYLLSELNAA
FLFPQLLLSDHINADRTRSWQTYYDALEELAVDGKLMLPGTPRGTKHNGHIFYVKVSNQVVRNELMTYLKHNGVSTASHY
IPLHSSDMGKRYGFFHGEDIHTTSESEKILRLPMFYGLDQGDIHYTVSLIKQYFKVPGEA
>lcl|NZ_CP026520.1_prot_WP_042231616.1_248 [locus_tag=PC41400_RS01295] [protein=hypothetical protein] [protein_id=WP_042231616.1] [location=complement(299156..300688)] [gbkey=CDS]
MKRVPDWLCAAMLSFLLLGIMFVLQGNIGINLADEGYLWYGAIQTSNGEVPFLDFQSYDPGRYYWVALWFKIFGNTDLLT
LRIAVLLFQMLGLTCALLVVRRMTKSWRYLLIIGVLLLLWVFPRHKMFDISICMIAVYMVSLLLEKPKPSQYLYSGITIG
LAAFFGSNHGLYVLVSFVLACLYIFFKLDRTNAIKKICIMVIGIFIGYSPMLLMFSFIPGFFNSFIDMIIWLFRLGNTNI
PLPIPWPWRVDFAHLQGIDILSGLIVGCLFLLLLLFPAGWVIRRFVSEKTNLFLTAAVFVGIPYAYVAFSRADIGHLAQG
IPPLLLGICFWALEYFKHNRAKLIVFMTAMLIVSCLSIGRFQPLYQQIKAAPNPYEKLTLMNHDVWIDASSAHFIRAVED
IKQQYLKQGENLFIAPHTPALYPILNLKSPLWDIYLLFPETEDKQRKQIEELKEKNAALVILGDIALDGRDDMRFRNTHP
ILWQYLISHYQQISAEAMPPNYTVLRRLDR
>lcl|NZ_CP026520.1_prot_WP_042231618.1_249 [locus_tag=PC41400_RS01300] [protein=hypothetical protein] [protein_id=WP_042231618.1] [location=complement(300979..302529)] [gbkey=CDS]
MTDNKKERYRFSEAPIWNIQRKYYEELGTKAWNNDQVPQYITSNPMIAAAYAEMIFGFLHDRAIKGCGSEPVTIVELGAG
AGRFAYHVLHELCQMRDYAGIPLPGFRYIMTDLAMSNVMAWKEHPALQSFIAEGLLDFAQFDAVHDTVLNLVVAGTTIGE
GDLEQPVIIVANYFFDGIPQELIYVSEGQIYEADVDVEYPEHKDSLKPSEWLNQITLHYEHRRAPEYEQETYPYRDVISL
YREQLEDSHILFPAAGLTCLQRLNSLSQAGFLLITADKGDHLLDNWKFAEPPELVLHGSFSLTANYHAFVHVFEQRGAEA
LFPPHHYININVGCILHLDKPKDYPNMRLAYRRCIERFGPDEFFSMKGWVDRNLDSMRLEQILSFWRLGGYDAEFFIQSA
KQISSLLPDANDEEKQDLLRGIELMWSSYYVMEQRYDLALDAGLILFEMDMYEHSKRYLEISVEEEQDEVVSTVFYCLAI
CCFELELVEDALSYIRKLLELEPDHEEALALISHFE
>lcl|NZ_CP026520.1_prot_WP_042231621.1_250 [locus_tag=PC41400_RS01305] [protein=pentapeptide repeat-containing protein] [protein_id=WP_042231621.1] [location=302817..303455] [gbkey=CDS]
MSKKQFITRWDDEQLAEANRSLAAVTGKSNLHKKDRLFPSSPFGQTAAGLEDFRGVELTETIQYLTVQGVDLSYARFVGA
ASLNTSTFTNCCFDRVKLDSRYVTREFSRCSFRGAKLNNARISERFEDCDFTGSNLSKAIANDVSFIRCRFSDVNFRGAM
FMHCRFEECSFEGAVFHNGSLAGSRFTGETGLLPVWGNTILDGVKINGERLV
>lcl|NZ_CP026520.1_prot_PC41400_RS01310_251 [locus_tag=PC41400_RS01310] [protein=VanW family protein] [pseudo=true] [partial=3'] [location=complement(<303511..303737)] [gbkey=CDS]
MANSDPILQANKINNLRIALKQTDGIVIYPGETFSFWKLIGKTTKAKGYIEGLMLSNGEVRTGTGADYANWRICC
>lcl|NZ_CP026520.1_prot_WP_042231625.1_252 [locus_tag=PC41400_RS01315] [protein=DUF6198 family protein] [protein_id=WP_042231625.1] [location=complement(304009..304707)] [gbkey=CDS]
MQPTNAAPAELQGSSLTKRWILYLVGIYILTLGVSLAIRAGIGISPQSSLTRVMTVIQPNLSQGTYNFLLELFMLLLTYL
VLPKDFKLKNFASLIPALALAFFLDFNLSLTGFIKLDVYYTKLLLLVFGDAALAFGLFLMIQANLVLMPIDMFVNTVFKR
TGKKWGNIKTVFDCSLLVLAACIGLIFLHKIMFIREGTILNAILVGQYIKLYTYLLNRSRAKKGLDPKVVKH
>lcl|NZ_CP026520.1_prot_WP_042231627.1_253 [locus_tag=PC41400_RS01320] [protein=adenine deaminase C-terminal domain-containing protein] [protein_id=WP_042231627.1] [location=complement(304918..306645)] [gbkey=CDS]
MKQTDANHLAKVHAIFDVLNGKEEADLLLKNLNILDVHGETVYQGSILVYDKRIVALNPDESIIRVKEVFDGQGLYAIPG
LIDAHLHFESQLANPTALAEAMVPCGTTTIYAECLDLLSAAGENGIQAARDLFKDYDKLPYRLYAFAPGKKTAAEVTKAV
LDMEPVIGLGEFEHFTYSSGDEDDFRKAAWLKEKGGFMNGHWGVTALSDMILNYLPAIGVSNNHDVWTGDDIEKSIRYGL
ATQIKFGVGSSEVIKILLRSIVDRKFPTDNFMLCTDNISVERLLKMGHMDWIISLCTEMGIPPIKAIKMASYNTAKSFHM
EDQIGSLTPGRFADIVLTDSLSKINPLYVFKDGELIAKDKKMLKNAEIDYSSMRTKGKPGLDDLTPDQLDIVPLEISEDG
TQAKVYLFDVYGRGHAKFYHEIWVTLKDGKIVPEYEGEKYSRLSVIQRYASGKRHVVNGLFKGVSIERGAVATFWPAPKP
YFVVVGQDSEEMCCCAKEVDQYSGACIVTENKSKKAVMPLEIYGVMADMTVDELTSSADAIDAALEEFGNRNEGEPVVNK
LLSLFISLHRFRFME
>lcl|NZ_CP026520.1_prot_WP_084706638.1_254 [locus_tag=PC41400_RS01325] [protein=LacI family DNA-binding transcriptional regulator] [protein_id=WP_084706638.1] [location=complement(306895..307875)] [gbkey=CDS]
MATLKDIAKHANVSACTVSRYLNSNIVVKKSTEARIQQAIKELGYIPNVVAKALKRNETSNVAVILPKINNLYYSEITSG
ISQTLSRHHYNLFVYEVDNQNLNEDDILQLMRENMIAGIIFIGLFSDKSFQESIQNVLEWEIPVVYANRSIPYQGFPLVY
PDLVKAGKLGAEHLYSKGKKKLAIVHKHLPEPLLDYFIEGFGQAGNGSFEGPLIMVIEEGKDLPDNCVFTLLENKIDGVF
VLNELSAIYLTKALHKSDIRIVHDIAVLSLGNSLMSEISTPELSCVDLQNRELGVISAEMILGQIRKQEIEPVTVLEPLI
IQRNST
>lcl|NZ_CP026520.1_prot_WP_164977107.1_255 [locus_tag=PC41400_RS29620] [protein=S-layer homology domain-containing protein] [protein_id=WP_164977107.1] [location=complement(308242..311835)] [gbkey=CDS]
MSIKGLKKSLAIFLTIMVVFQTFLSGIVVSAAGENPVSDAASVKLIDKGSSWSYSDEGLDYGYAMKGPFDFSKWKAGAAP
FGYKVTGGVTTGDVAPQTDFGKVATLLSFGGNEKKKHITTYFSKTVPISDTGVYETYKATLGVDDGIVLYANGIEVYRAG
MSVTEATYTTYATSNKTDPVVYKEDLSVNLKAALKNGDNVITAEIHNSSDSSSDLYFDMELTATIKQSGSGGTGNGPGTG
AADGKNIVLLPPNAAWKYLDNGTDQATAWRGEQFADSAWSVQAGPFGYPAGKATDAFGSVKQPISYGNNASMKYRTSYFR
TTIDVKNLVDYKKIAGTFGIDDGAVLYVNGKEVYRFNMPEGEPNFQTLSSTTINEPATETVDLTAALNGVLHNGINVLAA
EVHQRSDSSSDLYWDMKLVANPDETPSGGGGIKGPVVPTAIALTFNGDPKTSQGFGWYTDPSITGTKLEVAESAKVVSGT
FPAGDARTFEGTALPVSVYQSSSDKSSGKFTTYASHKVTATGLKPGTAYSYRIGDGQDGHWSDIRSFTTEGAGNSFTFLY
TTDPQGTTESEYVTWKHTLEEALGKFPASKFVTITGDLVDNGDIENQWMWLLNQPKEIFSKLPIVPALGNHESKANNNFT
YHFNVPNTSNTEAKPDGSVYAFDYGPAHFMVMNTEYNEAKGIDIVYKKQEEWLRAEAAKTDKKWKIVMFHKSPYSVANHT
NDSDVLFFRKKLTALFDELGIDVVLSGHDHTYTRTYPMFDNVPEKTSTNENNQIINPKGTLYLVSNAAGDKRYTPKAGPF
PFAAKYGQPGKEMFTGITVEDNSMSFQVYTTTEKGSTDLYDQFSIQKTGAKPNPVRDARMSTIQNGQSTLTWNAPQGGPE
PEAYRIYEKNDQMGANWNARVVPAAGFATYALTNLDPSKTYRFVIKAVSGKNSSDEVIAAYGNSGGGNGGDGGGSGSGGN
NSGNGNTGGAGNGNSGSGSGNGNSGSNSGNNGSTGGTGNSGNNGNTNAGGGETTPKPQVELTDIKDHWAYSVITQAVEKG
FIAGYEDGSFKPDKQINRVEMVSMLVKALRLKGAGASLTFTDAELVPDWAQAYVSQAVSAGLIDGFSDNTLRPNQSLSRL
ELVTLIVRASKIAVDPKAEPSFSDADKIPSWGAPYVAAAAKAGLIQGRDNNEFEPMATATRAESATMILSLLKHLKL
>lcl|NZ_CP026520.1_prot_WP_042231630.1_256 [locus_tag=PC41400_RS01350] [protein=S-layer homology domain-containing protein] [protein_id=WP_042231630.1] [location=complement(312135..312365)] [gbkey=CDS]
MNDIKGHWAQEPIEKFVQAGIIGGFGDGTFKPDRFITREEMVVILSRIVNLQGAGHAEDSNADPIDLAAQAGIIAE
>lcl|NZ_CP026520.1_prot_WP_042231633.1_257 [locus_tag=PC41400_RS01355] [protein=MerR family transcriptional regulator] [protein_id=WP_042231633.1] [location=312557..313378] [gbkey=CDS]
MLKISEFSKLSGVSVKALRFYNSLKLLEPAAIDPANGYRYYTQKQLLTVKRITAFKEHGFTLDQIKNFLEEHPIPDVKLK
LVDKKKELERSILEAQQQLKEINNRITRVEEADSRQQETPITVRNVQPQLTASIRDIVPRTHLCLLLDELTRYVNSYGED
EGRDLTILWYDHDNSDTDQADLEVALPLTKAIPSSGRVNVGLMPGLEAAASLVHLCNPYDNSCLAADRLVSWISSQGYVR
SYKEPIRETYLTSDKDIYGKMRLAELLIPLELG
>lcl|NZ_CP026520.1_prot_WP_042231636.1_258 [locus_tag=PC41400_RS01360] [protein=SDR family oxidoreductase] [protein_id=WP_042231636.1] [location=313509..314273] [gbkey=CDS]
MKRLEGKIALVTGGSRGIGKAISLRLAQEGALVIVHYGSRRAAGEEVVRTIEAQGGRALAIGANLETVAGVTDLVQTMEQ
TLVSLTGQNRIDILVNNAGIGTSNTLEETTEEDFDAIFAVNVKAPFFLVQQSLRLLRDGGRIINTSSGVTRIAYPHIMAY
NLTKGAINTFTLHLAKLVGSRGITVNAVLPGIVDTDVNAEWLHTPEGRQHAAEMSALGRVGEPSDIADIVAFLSSSDARW
VTGQMVDATGGSHL
>lcl|NZ_CP026520.1_prot_WP_042231638.1_259 [locus_tag=PC41400_RS01365] [protein=SDR family oxidoreductase] [protein_id=WP_042231638.1] [location=314432..315502] [gbkey=CDS]
MKSEENLSRKTALVIGANGVIGGNLIQHLMTLPEWDIIGVSRRGGEDTPQVRYVAADLLNEEDTRDKLGKLTEVTHIFYA
AYQDRPTWAELVPPNLAMLVNVVNAIEPIAPRLQHISLMQGYKVYGAHLGPFKTPAKETDAHHMPPEFNVDQQQFLEQRQ
PGSKWTWSALRPSVVCGFALGNPMNLAMVIAIYASMSKELGLPLRFPGKPGAYHSLLEMTDAGLLARATVWAATDERCAN
QAFNITNGDLFRWNELWPKIAAFFELEVAPPLPMSLDVVMADKETLWNTMVEKYDLVNNSFQDVSSWRFGDFVFSWDYDF
FADGTKARRFGFHDFIDTEKMFMDIFEDFRKRKVIP
>lcl|NZ_CP026520.1_prot_WP_042231642.1_260 [locus_tag=PC41400_RS01370] [protein=helix-turn-helix domain-containing protein] [protein_id=WP_042231642.1] [location=complement(315570..315941)] [gbkey=CDS]
MQEPDKTYNTAVEATLEVIGGKWKPVILFHLTFGKKRNGELMHLMPAVTQKVLSQQLKELVEDGIVSRISYNEMPPRVEY
ELTDYGWGLKGILHLMCRWGDMHIEKVYGDRGRVLFEPPQSDE
>lcl|NZ_CP026520.1_prot_WP_042231644.1_261 [locus_tag=PC41400_RS01375] [protein=Gfo/Idh/MocA family oxidoreductase] [protein_id=WP_042231644.1] [location=complement(316296..317411)] [gbkey=CDS]
MRSDKIGVGIIGASGWGKMAHIPALQALSEFEIIAVCTTRQKSAEEIAQSFGINHFYTNPYDLVANPDVDLVVITVKVPD
HNELVRAALDAGKHVYCEWPLALNTAEASDIFRLAAAKGVRHITGLQARANPAINYVKDLLSEDYLGKVLSINVSYSMPG
FPTKGSTIDQAHAYLLDKKNGADQLTITTGHLLDGITYLMGSPFTKISASLETQYKEISVMETGEVIQANAPDYVAINGM
LKSGALLSGQIRNSKIAGLSIEIIGTKGEISMKSKDGLMFQWDAFNLQAVLGENQTFQSLSVPSHYYHINPQEVNLISYN
IAQLYTQFANDLRKNTYTTPDFQTGCRLHYLLDQIIQSAETGTVQYLTVEE
>lcl|NZ_CP026520.1_prot_WP_042231647.1_262 [locus_tag=PC41400_RS01380] [protein=helix-turn-helix domain-containing protein] [protein_id=WP_042231647.1] [location=317581..317961] [gbkey=CDS]
MKDIANRTYNIPVEATLDVIGGKWKTLIVCHLTYGPKRTNELSKSMPAITHKMLTQQLRELEEDGIVIRTVYNQVPPKVV
YELSELGWSLKPLLNFMCDWGEKYIQELSRNGLPSQTTTASRGEEL
>lcl|NZ_CP026520.1_prot_WP_042231649.1_263 [locus_tag=PC41400_RS01385] [protein=GAP family protein] [protein_id=WP_042231649.1] [location=complement(318197..318853)] [gbkey=CDS]
MGNELILIVGGLALLDALSPATLGVTVYLLLTERERLNSRLMIYLLTVAGVYFVVGVSLMLGLDAVMASFSSIFQNRIVS
WVMLIIGGILFIASFYYPKSKISDLPKPKSKSMASMVALGFTTSLIEVGAAFPYFAAIGLMTTSNLAVFQWLPILAGYNF
IMVLPPLIVFTLHVLLGRVMQKPLESIRTRLSKYSGSAMSWIMCIVGLILIFNSLDYL
>lcl|NZ_CP026520.1_prot_WP_042231651.1_264 [locus_tag=PC41400_RS01390] [protein=MerR family transcriptional regulator] [protein_id=WP_042231651.1] [location=complement(318868..319623)] [gbkey=CDS]
MIHIKEVAKQTKITVRTLRYYDQIGLLTVSSKTEGGHRLYTEEDLKRLQQIQFLKGIGCGLKDIKVMLSDPKWNWSISLK
SQLVYIMEEQDKLKGIESSLRELINGMAVEGGDDYLALQKLIRLSTTNKSRLQSYRESMFNDMELELLKRVPKMRGEDPD
SLEWIALLGQIKRHMKNDPASLRVQNIVRRMLEKQMEDFNDEEEFINKLWDIRKSPMQSEKLGLYPIEQEVLDFMERAYE
IYISSLQKSTT
>lcl|NZ_CP026520.1_prot_WP_042231652.1_265 [locus_tag=PC41400_RS29625] [protein=hypothetical protein] [protein_id=WP_042231652.1] [location=complement(319649..319798)] [gbkey=CDS]
MKQGDIIIYACVIIGAGIGLMLDQAFPGVLVGLGLGYLIKMGLFKDKEN
>lcl|NZ_CP026520.1_prot_WP_042231654.1_266 [locus_tag=PC41400_RS01395] [protein=hypothetical protein] [protein_id=WP_042231654.1] [location=complement(320211..321155)] [gbkey=CDS]
MDYSILESGLSIISQCKERTGDIWEAHFGAAAIASYFFVKENVLSAKAVARITEDSHAMLRSRTYMHIPRNAEETDAQAK
EQAILTALEPTMDGLHWVGHNVIYAALSLWAIKELGHWGRKDDIEGISKLIESFHKTIPGRSWLGYSVSEVKKLEVSECD
CFPEIRNAEQLSDFILNELSSFRAIYKAEAHHDLIGHMLTFSQALNILHDLGHTELFLKGIPSLLKLIKVLRTSRDIDPK
HPPKLLSPVDRWPLQLANRSEWLPVEDEYWLRGFQHLDWDFGHTFKFPYSLYNHMNRLPRPNSKAVENFRYIIG
>lcl|NZ_CP026520.1_prot_WP_042231657.1_267 [gene=trpB] [locus_tag=PC41400_RS01400] [protein=tryptophan synthase subunit beta] [protein_id=WP_042231657.1] [location=321382..322545] [gbkey=CDS]
MIQEHLKEGYFGEFGGSFVPPELQEVLDYLSTQFNKYKDDPEFNEEFRYYLREYVGRENPLTYVKQLTDAWGGAKIYLKR
EDLNHTGAHKINNVIGQILLAKRMGAKRVIAETGAGQHGVATATACAMFNMDCVIYMGAEDTRRQALNVFRMELLGATVV
PVNKGQGRLKDAVDEALSDLVQNYQNTFYLLGSAVGPHPFPTMVKHFQSIVSEESKRQILEKEGRLPDAVIACVGGGSNA
IGAFAHYLDEPSVQLIGVEPDQAATLTEGVPSLIHGFRCLVLLDEQGNPKPTYSIAAGLDYPGIGPEHSHLKVTGRAEYV
TISNEEVLAAFQELSRTEGIIPALESAHAVAYAKKLAPTMSKDQIIIINLSGRGDKDVEQVFHMLKK
>lcl|NZ_CP026520.1_prot_WP_042231659.1_268 [locus_tag=PC41400_RS01405] [protein=DUF4304 domain-containing protein] [protein_id=WP_042231659.1] [location=complement(322772..323383)] [gbkey=CDS]
MLQQLFKDLIKQDVKPFLSKHGFAKKGLNFSKTTEGLIYLFNFQKSSGNSADNVMFYVNCGIYAAELAQIQSKEILTAPQ
ETECHFRARIGEIARAVPDRFAITPDTNRDDLRKTLLSGLEEVIHFYETMTSARSIVDYYISGPFLHLGEESFHLLLKSN
DAATAKHYLKALHEKYGAENRWTIFENKYKAIFNTYGVEFDKL
>lcl|NZ_CP026520.1_prot_WP_042231663.1_269 [locus_tag=PC41400_RS01410] [protein=hypothetical protein] [protein_id=WP_042231663.1] [location=complement(323471..324169)] [gbkey=CDS]
MLLKDTDQMDDLKEFLIDWYGKHESSYGVPVGELPAYLPKALHELYAFAGRWKDGSDDHLENSPEIFQRQDCLYSVERVK
RDGDKITFLEENQANWTCQVEAGNNNSPVYCDERLLWDDHAEGHTIVNVSLYHFLKSFCLQEVVFGCKYLYIVEGKLDPI
QMLFDKPIEDVWLNGYYISLKEEGPTHAFYRCGDVLVMERYGDFWLGSNSDLPAVLNDEVLSSIKLRKIKPD
>lcl|NZ_CP026520.1_prot_WP_042231668.1_270 [locus_tag=PC41400_RS01415] [protein=hypothetical protein] [protein_id=WP_042231668.1] [location=complement(324298..324762)] [gbkey=CDS]
MSYIVDFKDVSTVGLESSPVAEALAGLRANEARYFMNKYKHEFTVVPAGESQENLDYVNRILKEERDLAFAAKPLETSRF
QVENILFTYVFYEDGLALNVMYTVDNPKKRAVGFKLSEGMEVPQELEGKFKFARQKSKLAGTIRGSFFVIKGEY
>lcl|NZ_CP026520.1_prot_WP_042231670.1_271 [locus_tag=PC41400_RS01420] [protein=glycosyltransferase] [protein_id=WP_042231670.1] [location=complement(325028..326224)] [gbkey=CDS]
MARVLFINAGSEGHINPTIGVVQELVSRGEEVVYFCIGAYRDRMEKAGASVRTIDDQQFIEAFTSGGKDYVLERAIGLLL
TADVVIPTVLEQIRGEHFDYMIHDSMFGCGRMLAQILKLPAISSCTSFAHIEETFDALLEPPPVKTPEEIVERFQSLTKT
VGEKYGVEVRTPYEVFCNPAPFTIVYTTRDFQTNEEKFDQTYKFVGPSLSSRPAQKNFDLTAIKGKDPIYISLGTVFNQA
IDFYKLCFEAFGNTEHTIVMSVGNNTLISDLGEIPDNFIVKNYVPQTEVLQHTKLFVTHGGMNSTHEGLYNGVPLVVIPQ
NADQPMVAGQVAKIGAGLALQMQGLTAHQLREAVDQVLSLPSFQKAVAHIGESFRKSGGYRQAVDEIFEFKNQSPIYQ
>lcl|NZ_CP026520.1_prot_WP_042231672.1_272 [locus_tag=PC41400_RS01425] [protein=MDR family MFS transporter] [protein_id=WP_042231672.1] [location=326433..327953] [gbkey=CDS]
MRNKLSAIVGGLLLAILMASMDNTIVSTAIGTIVGEMGGLDKFVWVTTAYMVTEMACMPIFGKLSDMYGRKRFFIFGIIV
FMIGSALCGIAQSIVELSIYRAVQGVGAGAMVSIAFTILFDVVSPKDRGKMMGAFGAVFGLSSVVGPLLGGYITDHLNWS
WIFYINLPLGFIALLMVALFYKESAEHSQQKIDWAGSILLVGTVVCFIFGLELGGKTYAWDSPQIIGLFAGFAVLAFLLV
MVEKKAAEPIISFGLFRSRLYTTSNLVAMLSGAAYMTAAIFIPIYVQGVLGGSASNSGLTLLPMMVGSVVTATLGGFLMT
KLQYRTIMIGTLAMLLTGLALLSQITVESSRLAITGIMILIGLGIGASFSVLSTSVLHSFPAAQRGAASSTLNFNRSLGM
ALGITIFGMVQSHSLLSKLTESLGKGDIAVPGDLNLSDPHSLLNPAFRQSVDPQLLDTITAGLSGSIGATFAWSLIPAVL
ALIAAFLMTKDKMDPAAEGEEFATAH
>lcl|NZ_CP026520.1_prot_WP_042231675.1_273 [locus_tag=PC41400_RS01430] [protein=YafY family protein] [protein_id=WP_042231675.1] [location=328000..328962] [gbkey=CDS]
MKLERLMTITILLLNRKRVQAQELADQLEVSLRTIYRDLDSLGQAGIPIVSYTGMEGGYEIMDSFRLDRQLLSFDELTAL
STALRGLESTKAYDRSNMDLLLSKVGAMVAQAEQGRAGEGDRIHIDFTPWKNSEEDQSRYNSLRQAVNDRKLIRFRYTSR
TGDEQKREVEPMALVLKNYAWYLHGYCRLRGDYRIFKLTRIRELDIQAGTFIRRAESLAQLNDRWKKPEIHEANEAIAVV
LQFKASAAVSVMDHFDEKDIERLPDGQLIVRMTYPSERWLIRTLFHYMTDVIVLEPASIAAKVRQTALDIAGQYGVSDTK
>lcl|NZ_CP026520.1_prot_WP_084706640.1_274 [locus_tag=PC41400_RS01435] [protein=SDR family oxidoreductase] [protein_id=WP_084706640.1] [location=328988..329737] [gbkey=CDS]
MSLDGKVAIVTGGALGIGRATCIQLAKQGAKIVVNDFGNELTAESVVSEIVAGGGEAVSIPGDARSAGQMEQLVEAVKHA
FGRIDVLVCNASKGATAATFAELTWESFERQLTDELKAAFEPTKAVIPTMMAQQFGRIIYLSSTDGKDPTPQYIAFGTAK
GGLDTFARYIAQEFGPYGISANVVAPGFVRTDETFMISEEEARVIGSFTPLGRIAEPEDVAEVISFLAGDGARFLTGTYT
PVTGGLIME
>lcl|NZ_CP026520.1_prot_WP_241688771.1_275 [locus_tag=PC41400_RS01440] [protein=methyltransferase] [protein_id=WP_241688771.1] [location=330176..330628] [gbkey=CDS]
MLEPIPWNHVHTLVELGAGTGVFTQMIYDLVRPDCQVAVFEKEEEMRELLKREFCGFHMFSNALELKDSLSSIGADRVDC
IVSGLPFANFSQAEREKIIRQVRETLREDGLFITFQYSLQMKTMLEAAFDEVSISLVPWNIPPAFVYTCK
>lcl|NZ_CP026520.1_prot_WP_042231684.1_276 [locus_tag=PC41400_RS01445] [protein=DedA family protein] [protein_id=WP_042231684.1] [location=330697..331305] [gbkey=CDS]
MSIDSILDAIHHFGYFALFFSLWLGIVGMPIPDEVVVMTGGMVGSLHLLQPVPAYMITYLGVVSGLSLGYILGRVMGPLV
VERLRKKKKIAPYIQKSDRLLEKYGSFALAISYCFPVVRHVLPYLAGVNKMPFPKYALISYSTGLLWTTIYFIIGYFFGS
NIERIGNNVYKYGYYGLAAAVILMIGAVIIQAIRKKKKVCSE
>lcl|NZ_CP026520.1_prot_WP_042231687.1_277 [locus_tag=PC41400_RS01450] [protein=response regulator transcription factor] [protein_id=WP_042231687.1] [location=331302..332003] [gbkey=CDS]
MKNGKGASVLIIDDEPEIRDIVQIFLKNEGYEVMEAEDGAAGLELLRKKEFDLIILDVMMPRLDGIQACMQIRQTRNTPI
IMLSAKNEDIDKILGLTSGADDYLSKPFNPLELVARVKSQLRRVRMSETQDKRQPLELSIGDLVLNIPRHQVTIQGKDIS
LTPLEFDILELLATHRGHVFSVDKIYETIWKDPALQSDKTVMVHIRNLREKLKDNPREPKYIKTVWGVGYKID
>lcl|NZ_CP026520.1_prot_WP_042231691.1_278 [locus_tag=PC41400_RS01455] [protein=HAMP domain-containing sensor histidine kinase] [protein_id=WP_042231691.1] [location=331996..333420] [gbkey=CDS]
MTRLKRPDWIKSLQVKLMLAFGAALVLTTILSLLFDYVTRESRKVYFTSFETSKQNVKQLLSDMERQIPVLSGPEGIDPL
LKRESEASRLRILLSDMQGTIVYHSPSVTEDKIDLYQIIYEQRETENQPEPGKMYTAITPIQYQGQKLFLVVSGPLFAEQ
GYIYENPRLYSSLVFLLLFLLLFYWFTFSKMKEIRTMSRGVQQIAEGKLSIRLPVKSKDELGTLTASINEMAGQLEEMIA
KERLTEQAKSELITNISHDLRTPLTSIIGYLTVLIEHPETGGALKNYVHSALNKSNQLKKLIDDLFEYTRLTSDQVTLDK
SRVDLVGMLHQIIMEFLPLAEQHQVRVNSTLPDRKVEVVIDADKVVRAVDNLLTNALKFSVKPGSIDIVLNAEHPWAVLS
IANNGLAITPEQEKRLFDRFYKADESRNDHRMPHGSGLGLSIAQSIVQLHEGDIWLRRTGSYYEFFIRIPLRIY
>lcl|NZ_CP026520.1_prot_WP_042231694.1_279 [locus_tag=PC41400_RS01460] [protein=PadR family transcriptional regulator] [protein_id=WP_042231694.1] [location=333782..334147] [gbkey=CDS]
MADVNETITGLLSELRRGTIVIGVLSQLSEPRYGYSLVPLLEEKGIAVDPGTLYPLLRRLEKQGLLESTWDTNESRPRKY
YLLSPFGKKVYEGLFQEWRKLVANMEGILTDLGGGEQHGSN
>lcl|NZ_CP026520.1_prot_WP_042231697.1_280 [locus_tag=PC41400_RS01465] [protein=membrane protein] [protein_id=WP_042231697.1] [location=334134..335171] [gbkey=CDS]
MEVIDRYIYAVTQRLPEQQREDIKRELQSLIEDMLEERMTTGPASKEDVESVLLELGPPNELAAKYRGYERYLIGPMLID
AYLTTLKIVLASIVIGLTAVFIIDTFLSHSGKLDPVTSYLASLVTGAAQGFAWVTVVFAWMEYRQRAGNGSRNKTWRPSD
LPLIPDAKLKINRIRPILSIFFTIAVMMICLYLTDLLGIWRNRDGVFTSVPYLDNDVFRHYWPLLWIVAALSITKQIVRI
IVRYRSSKLLAFHIVMTIITTMLVCIMLSDQTLWNPHFIRDLTASNLMPAEGDNFKTLVSTWPRVSEWLINAIVLFALLN
IIRESLAWYRAKSSSSPIPQKKPLV
>lcl|NZ_CP026520.1_prot_WP_042231698.1_281 [locus_tag=PC41400_RS01470] [protein=aldo/keto reductase] [protein_id=WP_042231698.1] [location=complement(335284..336147)] [gbkey=CDS]
MNQTVEAANGVNIPRLGFGLYKIKSGEDFERAVEEAIRTGYRHFDTAKIYGNEGALGRVIPKSGIPREEFFITSKAWTTD
LGYHTTRKAFEQTCRRLNVTYLDMYLIHFAGPHYVEAWKAMEELYDEGRIKVIGVANFEIRHLENLKKNSRISPLVNQVE
THPQFPQRELHDYMVRHRILHEAWGPLGQGNEALLKHPELSAIAHRHRKSVAQVILRWHLERGIIVIPKSSNPQRIKENS
EIFDFELNEEEMERIRRLDTGKRYSVSPTGYMVNPTYVKLMKLFICR
>lcl|NZ_CP026520.1_prot_WP_042231701.1_282 [locus_tag=PC41400_RS01475] [protein=metalloregulator ArsR/SmtB family transcription factor] [protein_id=WP_042231701.1] [location=complement(336144..336527)] [gbkey=CDS]
MDQERAAAIAIEFKNNQKVLNAIGDETRQAILMALIQGTQNPGMRVGEIRVKTHLSRPAVSHHLKILKEAQIISVRKEGT
LNYYRLDSASNLKLLKNLVNEIEKALAPCEEQASDQRTRLTGRGDRE
>lcl|NZ_CP026520.1_prot_WP_042231704.1_283 [locus_tag=PC41400_RS01480] [protein=serine hydrolase domain-containing protein] [protein_id=WP_042231704.1] [location=complement(336838..338301)] [gbkey=CDS]
MFIVFKKMGLSLLCFLLSAVSFMAEPASAAEVQNKAEIQQKINNYVTKNMKANHIKGAALAIANNDEVFYAQGFGELSDG
RVIGGTTPFPIASLSKSFTALAVLQLVEKGLIDLDAPYVSYFPDLSPKDKRVRDITIRNLLNQTSGLNDKVNPDMTKFPQ
YQSLQEINRSLDAVRLAHDPGTAYSYYNPNYEYLALLVERISGQNFSDYLANQILKPLGMSQTVNISSTQQINDNPAIPR
GHYLVLGRPVSQDEPSWFVEGPAGIVSTAEDMAKWMLAQYSGRLLSPELMELYHEAGQMSSYGMGWLAGADEQEGRTISH
SGIFWTYKSEEIIYLDQQMGITMMFNSGLNMFVDYAAFTDGIVKIIHGQDPDASILNSSNLEIVMILLIIATIFWGVSGL
IRLMRSNKSITAAKLIISTIGRIIPVFILLFLSPIATFIGGGRVLPWFGLWTTMSSLILWLAVLSLINIAHLACRFRLYY
LSKKKGS
>lcl|NZ_CP026520.1_prot_WP_042231705.1_284 [locus_tag=PC41400_RS01485] [protein=sensor histidine kinase] [protein_id=WP_042231705.1] [location=complement(338349..339458)] [gbkey=CDS]
MAKIMNWLQLKRMSLLQSFIFLGCLMLIAVLAVIVVEFQWAESLRQRYAAHSEANDWILPSLIALVLLTVAVGIVLMASL
FYRWKLRKPLDILMKASEKISSDDLGFRISYDGKDEMGELCRAFEIMRGQLEKNYTTLWRSIEERKQLNAIFAHDLRTPL
SVLKGNFEFLSTYLPQNKISEEKLLDMIQTMSVHIVRLEKYTEAMSSIQKMEDMPIHSHSIEMDQLIALLNESARQIIGQ
YGRTFNTSIASDSQSISVDTHLVLQVFENITANAARFASGLVRIHYCVKMGYLSITVLDDGTGFSEADLHKAMLPFYRGE
VSNANEHHGMGLYTCKILCEKHEGNLQIDNGPSGGGQVTASFLTGLINS
>lcl|NZ_CP026520.1_prot_WP_042231708.1_285 [locus_tag=PC41400_RS01490] [protein=response regulator transcription factor] [protein_id=WP_042231708.1] [location=complement(339443..340138)] [gbkey=CDS]
MEKHRRHTTKILIVDDEPDLRKLLTDYLEINGYVVMAAQNSHEAIRQIEEHPDLILLDINMPEMDGLELCRKIRNFISCP
ILFLTARVEDTDKISGFRAGGDDYIQKPFSIHELGARIEAHLRREIRSQHKTSVKFSEDLVIDYSARALYFRDERIPLAK
KEFDIIELLSTHPGMVFDKERIFEKVWGIDDEGDSAVIVEHVRKIRYKLKEHGCDDRIETVWGVGYRWRKS
>lcl|NZ_CP026520.1_prot_WP_042231709.1_286 [locus_tag=PC41400_RS01495] [protein=TetR/AcrR family transcriptional regulator] [protein_id=WP_042231709.1] [location=complement(340340..340924)] [gbkey=CDS]
MQKESARDRILRVASELFYMEGIRAVGIDRIIKESGVAKASFYRSFATKDDLAVAYLESRDELKVEKLERLRQLHPNSPK
EQLDTLIRDVTEQMGMPDYRGCPFLNAAVEFPDPDHPAHIKLLGIQRSFWSRVLELAREGGAREPEELTAQLRMLYDGFV
MKSYLDRSEFDPEYFRNAAELLLEKQLAPDNHRG
>lcl|NZ_CP026520.1_prot_WP_053228751.1_287 [locus_tag=PC41400_RS01500] [protein=MFS transporter] [protein_id=WP_053228751.1] [location=341096..342565] [gbkey=CDS]
MPNTLSDASAQVVRTRDPVSKGDGRRSRLMLPALLLPVFMAVSNAFIVNVATPSIQKGLGASFADVQFMLTGYTLTFAVL
LIIGARLGDRFGRRRMLFIGVALFTIASLLCGVSAHVTLLIASRVVQGVGAALFMPQILSLIQANYPPERRGAIFGMYGA
TQGIAATAGQIIGGLLLRMNLWDLDWRAVFLISVLLGALILALIPFIPESGSPDRARLDWIGAALAAAGLLMLVYPLVQG
QRQGWPAGLVVSLLLSAPVLVLFAWYERRLLRRDRLPFMNVDLFRHKVFTSGILVVLLLMSTQGAFFLVSAYMLQLGLHF
SALQAGLVIGSMGIGYFLASLFSSRVTAKLGAHVLTVGSILTTAGYLLLSWAVRTTGISFDIGVWIPALAVLGIGQGLVA
APLTNIVLAKIRTTDIGSASGVMTTSIQIASTIGIALIGIVWLSALEAHTGSVSAYPDAFNICLNVLAAVTVLILPLALM
LAGRNGAQK
>lcl|NZ_CP026520.1_prot_WP_042231712.1_288 [locus_tag=PC41400_RS01505] [protein=DUF3139 domain-containing protein] [protein_id=WP_042231712.1] [location=342918..343292] [gbkey=CDS]
MRIIKRTILLVISLLFLFIGSIIVFMNLRAEKLENEVRSYLTSKGYEEHQIQKIDGKISIQPVYQVVVIFTDEPNVTYCY
GSKKYYSILPSKGEDKVFQLSVTRFIDQNYEYKHIEPNVKEQLK
>lcl|NZ_CP026520.1_prot_WP_042231714.1_289 [locus_tag=PC41400_RS01510] [protein=hypothetical protein] [protein_id=WP_042231714.1] [location=343456..343947] [gbkey=CDS]
MKKKLTVIAITVGIIGCSLLTAGVAFGDVDPIKNLNNLKSAFESNKKKLNEMPNQTPEEIEATVRAGKEPKDLGKEVAKK
EQIADPDGEAKFANQLKSYIAGMKDSVSELKHMAEETKDSKYLKKAEELEKKIARYEQEQQAYLKKEGKTVKELRKELDI
PAY
>lcl|NZ_CP026520.1_prot_WP_042231717.1_290 [locus_tag=PC41400_RS01515] [protein=SMI1/KNR4 family protein] [protein_id=WP_042231717.1] [location=complement(344150..344647)] [gbkey=CDS]
MDKKMGEFIIWAKENGWDIIEKSGFQLNLDSSIVSRYKEIPNEYLNFLSVVEKCMTPNEKTWFICEDEFNNSSDIAFKWN
EYELLSLEAAVNDDSWKSEITAWWDHYLPIVMSVDGGYSFYAIDLTHDKGAIVRGYEPEFEEVDKVANTLEEFLGLIMSN
SIEFQ
>lcl|NZ_CP026520.1_prot_WP_042231720.1_291 [locus_tag=PC41400_RS01520] [protein=DUF421 domain-containing protein] [protein_id=WP_042231720.1] [location=344851..345543] [gbkey=CDS]
MTYGEILFRTAASFVALLVLTRLLGKKQVGHLTFFNYVTGITFGSTTAEIIVNRQITLLQGISSLVLWSLLTFIIALTGL
RSVKAREILDGQPTILIKKGMILEKALAKQHLNIDDLSMLLRNKDVFSPEEVEIAILEPDGKLSVLRKEELLPATKKDIK
ATVTPVQKIPTELIVDGKIVEKNLIKMDVSAAWLNQQLKKAGIHSLDQVFYAELQLDGTLYIDKYQDSMP
>lcl|NZ_CP026520.1_prot_WP_042231723.1_292 [locus_tag=PC41400_RS01525] [protein=hypothetical protein] [protein_id=WP_042231723.1] [location=345941..346465] [gbkey=CDS]
MRRLKVLLFLFITGVVIVGCNSQSKSGNYETVTFNGLEFSIPKEWTQTPIENSKDQISYKPDFGKDKYGFFYIYSYPYPR
NVYKSKFIDWEKNTYKGIFALSDETKLNKNMTVLTYKTNFNAIKNAQLPDQVVQIIVIDDVDRVLEVSMDIPRDYFEQNP
SIFKHIQDSLTISK
>lcl|NZ_CP026520.1_prot_WP_129112441.1_293 [locus_tag=PC41400_RS01530] [protein=hypothetical protein] [protein_id=WP_129112441.1] [location=complement(346791..347018)] [gbkey=CDS]
MAQKLAVIDSAVSCIIDSGNAKSKGKTEVYLHLANGEVVAGTSFPVMDEPLAGGAFSLAITPISSAFCQVITLTE
>lcl|NZ_CP026520.1_prot_WP_042231725.1_294 [locus_tag=PC41400_RS01535] [protein=alpha/beta fold hydrolase] [protein_id=WP_042231725.1] [location=complement(347072..347992)] [gbkey=CDS]
MEMNVSTSTPVISIKPVVLSAPGRGEDLQVRVSAPAMGSQLPIIIFSHGFGWSLDGYGPLVDHWAARGFVVIQPTHLDSR
TLNLPPDDPRTPLIWRFRVEDLKLILDQLDHIEASVPGLSGRLDRSRIAAAGHSWGGQTVSMLLGARVLDAHGEPGENMS
DSRIKAGVLLATTGLGGDDLTPFAAEHFPFMNPSFSDMRTPTLVVAGDHDQSLLSTRGPDWFTDPYFLSPGSKSLLTLFG
AEHSLGGIAGYNVAETTDENPERVALLRQLTWSYLRDALNLDDSSWLAARKALEENANPLGRIESK
>lcl|NZ_CP026520.1_prot_WP_042231728.1_295 [locus_tag=PC41400_RS01540] [protein=MarR family transcriptional regulator] [protein_id=WP_042231728.1] [location=348102..348545] [gbkey=CDS]
MDKSELNKEELRLWYMWKGSFHSIFGRVIKEMSEHTGLSEGDYGILDRLDLLGDGSLRQQELAESMDWDKSRLSHHLTRM
EKRKLVMRKPLDTDRGVQVIITTVGKSALDDARPIVSKAIRKHFLDQLTDQDIESITKLAERTKNRG
>lcl|NZ_CP026520.1_prot_WP_042231730.1_296 [locus_tag=PC41400_RS01545] [protein=hypothetical protein] [protein_id=WP_042231730.1] [location=348703..349128] [gbkey=CDS]
MVRILDKAAVQPRSRFNPAKSFTIRRSPQKSGIATIPIRIPANSQPNRVDLVASVGVRGVTGIGQILFRVFRGNTEIFNT
QQGIESTGSEQNYIVTFQAEDRNVKAGTHSYTVTAENRTANTRVDVVGPISFSGLAVKTRN
>lcl|NZ_CP026520.1_prot_WP_042231732.1_297 [locus_tag=PC41400_RS01550] [protein=cell wall hydrolase] [protein_id=WP_042231732.1] [location=complement(349392..349835)] [gbkey=CDS]
MAVVKARDQDINMLARLLRAEAETEGEMGMLLVGNVGINRVRGNCSDFKGIRTIPQMINQPHAFEALQHGLYYQNARDRE
RRLARRAVNGERNWPGKFSLWYFRPGTFENPGPCPETWYNQPLAGRWKKHCFYEPTAEECENIYNTF
>lcl|NZ_CP026520.1_prot_WP_042231734.1_298 [locus_tag=PC41400_RS01555] [protein=GntR family transcriptional regulator] [protein_id=WP_042231734.1] [location=350065..350808] [gbkey=CDS]
MEVKPEHPAKAKKPLYLTVYNELFKKIMSGVFPSNSQLPTEPELAKMFDVSRMTLRQALALLQDDGLVKSIHGKGNFVTP
SRMVQRNDGLDKIGNPIYKCHTEDIDHVDIQFRLDLESDYTKEVLKRKATAVVAVERWYMSKGQAVAFGFTFMAIEAVTE
LNLDLQNEEQLLDMLENKIYELANSATVEVKHSTVMNTSSQKFELDADEGCDLLLESLYVNEQYPIVYNKYYIPKEFSRI
VINASKK
>lcl|NZ_CP026520.1_prot_WP_042231736.1_299 [gene=deoD] [locus_tag=PC41400_RS01560] [protein=purine-nucleoside phosphorylase] [protein_id=WP_042231736.1] [location=351017..351733] [gbkey=CDS]
MSTSPKETPHIQPGGTEIAETILLPGDPLRAKFIAENFLDNVVQFNEVRGMLGYTGTFKGRKISVMGTGMGTPSMSLYSW
ELIHVFGVKNLIRIGSAGAIQDHLNLYDIVFAMGSATDSNYGHQYNLPGHYSITASFELLEKAKKIADAQGQKVHVGNVL
SSDIFYNADPTALKRWSEMGILCAEMESAGLYMNAAHAGVNALCILTISDHIFRHESTTPEERQTAFTKMMEIALELA
>lcl|NZ_CP026520.1_prot_WP_042231739.1_300 [locus_tag=PC41400_RS01565] [protein=tetracycline resistance MFS efflux pump] [protein_id=WP_042231739.1] [location=351764..353101] [gbkey=CDS]
MATRTEQTPQTTPAFQPKRAVPVILLFFVFALIVDNSFKLVSVAIADDLGISAAAVSWQASLAGLVIGIGAVIYASLADT
ISIRKLLSAGIILICIGSIIGFIFQHSYLLVIIARMIQTAGLAAAETLYVIYVTKHLPKEEQKKFLGLSTSSYSLSLVIG
SITGGYVSTYLSWSTLFLIPLLSLILLPFILKYLPKEQSRKSHVDFIGLLLISAIATSVMMYLTDFNWIFIVIFAAAAGL
FLGYITKSKKSFIGISFFQNKRFAAVLAIAFVIYSVQLGYIFIFPFLLQKMYNLQLDTISLLLIPGYVTAILVGSLSGKI
AKVLTNKQAVSIAMLIIVCSLALPGLIGGSVIVYVLSMLLFSGSFAFMYAPMLDSCIRSIPVERSGTAIGFYNLVLNVAV
SIGITYTASLMDVVSFGTVLWILAAITLFALLMYGLLVGRLTEPK
>lcl|NZ_CP026520.1_prot_WP_042231741.1_301 [locus_tag=PC41400_RS01575] [protein=hypothetical protein] [protein_id=WP_042231741.1] [location=complement(353364..354572)] [gbkey=CDS]
MPNDDQARPPAGSIKEDGRYPIDLTGPSSHTLVRQKGVGSLSIGPSHLGKKADLHVAPDSLIDWTVFDAFSTPAGSPWPR
FLHYTGSDAGFFDWAQKRPIEEMTWAPILSEDTVVNASPSILHGLTIELGPSGGHLNLKLPRKPFRLNVSGDLSRFSATG
NMPSSLTLAPRTGRRKKDTPFLMPDLGELHQVTSLALQNAPLGQPISLECLDRFPNLDSLSLWGNFCDLDLLARHTGLTN
LELRFMPDLEDLPSLNVWPLLDRFIAYNVEEFTGKRLKQQMKTRAKTRPWTGHASVSQLRKPEWWSTEFGRPFSSWPKRL
AKLANEAYDVAQENLAQARSFADAEAVITAFTLRFNTLKGIETTEREDLGEAVWQLSQSDHLIGQPITEEMAQSWFDAAR
DY
>lcl|NZ_CP026520.1_prot_WP_042231744.1_302 [locus_tag=PC41400_RS01580] [protein=DUF4062 domain-containing protein] [protein_id=WP_042231744.1] [location=354759..355325] [gbkey=CDS]
MRKKLQIYISSTFNDLIEERHTAVEAVLQAGHIPAGIEQFFKESPVNIRKRWIDESDVYILILGGFYGLTLPDESKSYTH
WEYEYAGETGKPRFAFVVTDEALRQKPYDFAAIEYYRKFQEFKQSVMEQIPIYYVEDVRHIKMVLRDQLPEYAARNDLYG
WVSGKDVPDVQKLLEENARLKAELEKKK
>lcl|NZ_CP026520.1_prot_WP_042231746.1_303 [locus_tag=PC41400_RS01585] [protein=S41 family peptidase] [protein_id=WP_042231746.1] [location=complement(355401..357104)] [gbkey=CDS]
MSVIGTIDSVKWLTVYELIMCIINLFVLTVYAVPIKKYYRWFDFLPSAGVLAAIASVLSGDMTLLALIIESLTLLVFLCT
IRRLLKPQAWTAPKKKAFRIVRNVICVCGAAPIVFALANAGVLRYNPVSDFSGMSYSKAFVAMNERLSREYPFGEWKNID
WNERRSKYEPIFKRAEKDKDSDMYYKTLREYLFSLRDGHIKIVNDNVYDGNAVFKKEVGGGFGLSTVQLDNGKVLVSLVL
ENSPAAKSGMKLGAEIVAWDGKTGREAFGRTTWSENNMATDEVKRFNQGRFMVRAPIGKEVQVEFRNWGESEPTRTKLTA
YDDQFETLKKTKTKLTQADLDASPIEGKILENGYGYVKIKHFLPNSNDTSPEKSLADLLKMFQDKHVKGLIVDLRNNPGG
DDQLAADLAGFFVKEMKHYEYVSYYNRYTGEFELNRNEVVKVKPAKPYFDGTVAILINGRTGSSGEGLPLVLKGIPNVKI
VGFTSTAGSFGLISSPIEIKMPEGYVLQFPDGRSLNQNKKVQGDADQTGVGGAVPDIKIPLNEETFKESVMDGQDVELKY
AIEAMEK
>lcl|NZ_CP026520.1_prot_WP_042231748.1_304 [locus_tag=PC41400_RS01590] [protein=HAMP domain-containing sensor histidine kinase] [protein_id=WP_042231748.1] [location=complement(357198..358565)] [gbkey=CDS]
MTRSLYVRVVLTFLAAMGIGLLVSVLAGLALFEEKIDRIGQNDLVAAGEEIIRVYEQTGPANLDRFMESMVKLTSYSVQL
YAGSGAMKSFKYSSDTDDLAISQESIRKVREGEIYRSSSKDKETFIGLKFQIGGERHALFLRPSSKNESTLIHLMLAILL
LVLVTGSICILIAAIYVVKPIKTLTAATKRLAKGDFDVELGVRRKDELGNLARSFNEMAVDLKQLEQMRQDFVSNVSHEI
QSPLTSISGFAKALKDMNAIAEDERIQYLDIILAESGRLSRLSDNLLKLASLDSNHHPLEYATFNLDEQIRRVVVNCEPL
WSEKNMDIDLDLPQAAVITADKDQLNQVWTNLLGNSIKFTPEGGLIHFQLRTRMDGYSVTLSDSGIGIAPEQLGRVFERF
YKTDRSRNRSGGSGLGLAIVKKIVALHQGDIYMESKVGQGTKVVVHLPAIPPDKS
>lcl|NZ_CP026520.1_prot_WP_042231751.1_305 [locus_tag=PC41400_RS01595] [protein=response regulator transcription factor] [protein_id=WP_042231751.1] [location=complement(358562..359239)] [gbkey=CDS]
MTQILVADDDAHIRKLITLYLKSEGFDTVEAANGIEALSVMENSTVDTVILDIMMPHMDGWELCREIRRLYPEIPLLMVT
AKGESGQKVKGFQLGTDDYVTKPFDPLELVMRVKALLKRYRIVSSQIIQLGGIVLDRRAHKVIREGKELILPLKEFELLF
KLACHPGQILTREQLIVQIWGMDYKGDDRTVDVHIKRIRERFAEDAEHFRIETVRSLGYRLEVKK
>lcl|NZ_CP026520.1_prot_WP_241688873.1_306 [locus_tag=PC41400_RS01600] [protein=EVE domain-containing protein] [protein_id=WP_241688873.1] [location=complement(359614..360024)] [gbkey=CDS]
MGVVSASHVQRGVQGGFAQMCHGKSAPLRRMQSGDWLVYYSPRTEMGGGEPLQAFTAIGRVVDDHVYEYAMSATFVPFRR
NIEYVPCREARIVDLLDRLCITRGKRNWGYPFRTGHFEIDREDFLTIAEAMHADLG
>lcl|NZ_CP026520.1_prot_WP_042231755.1_307 [locus_tag=PC41400_RS01605] [protein=DinB family protein] [protein_id=WP_042231755.1] [location=complement(360194..360790)] [gbkey=CDS]
MNTKEMLRKFEETVSGYIRELDGVSLEQLLWKPAEDEWSLGQMHVHLINSAQFMHLRNVAMCLAPNGNPAVSSTGKTKPG
EEVFGAGSFPPDRVQVPPSPQYTPPQPEGKEQLVDGLRDTVRRMTEIEPAVVAAFDPVAQAISESGKDTDFELVRNTVVH
PRFGGLNALEWFQLIEMHYRHHLLQRKRLDDAWRAAHV
>lcl|NZ_CP026520.1_prot_WP_241688772.1_308 [locus_tag=PC41400_RS01610] [protein=hypothetical protein] [protein_id=WP_241688772.1] [location=complement(361000..361539)] [gbkey=CDS]
MEYTGASDNRKSSSNGKKRNKHRFWIGITGLITFGMFMYFQGSTGSIKDVQSNGIGKGMKAGIQLAAADDSAGLEARDFK
IEMKKNTTTSRMLIWDFAAEDGDVVTVKVDGNILQANITIMHSPVALDIPIPSAVEIIGVKDGGGGITYGVKFPGAVQNN
AYFNAAPVGSSNVYTITGQ
>lcl|NZ_CP026520.1_prot_WP_042231763.1_309 [locus_tag=PC41400_RS01615] [protein=hypothetical protein] [protein_id=WP_042231763.1] [location=complement(361644..362087)] [gbkey=CDS]
MKVLKWLLAIVFYHSFMFGVIIATLFMPFIIYGDIKKILINEVPVASGGVIVISLLAFFIYLAMRSKFLGIPYRKITILL
PALQMLIYTSFALSVGVIILNKWADEGLYSKGWAITLLLLAIAAIRLCMSLLYWKYPVVRRTNRYKE
>lcl|NZ_CP026520.1_prot_WP_042231766.1_310 [locus_tag=PC41400_RS01620] [protein=AAA family ATPase] [protein_id=WP_042231766.1] [location=complement(362084..365761)] [gbkey=CDS]
MRINRRSMLKALKFLFPEYQLVELVREKGFIAKKEEPPAPTRREQPTQQYHAGSGARSAGPSYSEQAAAGQNISSSTSRP
VQNRAHTSQEEGRNEAQSALSPKQLDKLFREVEKNVNSQVIGQAFFVDDLIAGYKQGYMNGRQNNKPRNVILLAGAPGTG
KKTALEWLVREMHARRLISRPQLAEIDLQRYEADELSGNFISDMAEAFQHSEGTVLFQGIKGADPAIISLVAKLAAEGSF
RTKEGVRISADDYFIIFYIDEHAERESEYGQVPSVLARHLPTSILQTILAAAISSPLEQETMRQIAEILLGKAIDDLSRD
MQTEITVHPSTYEALADIAAANKTFGEAVQHWIDKELTVVLSGLRARNEIRGNESVRIRFKNDSFFVESNSLEIPIRALS
FVRQESIDDVLKELNALTGLEPVKKFVYELMETVQVNKLRARESGGSVAMSLHMVFAGNPGTGKTTVARLIGRILKALGL
LPQGQLIEVTRQDLVGQYVGSTAPKTMAQVNEALGGVLFIDEAYTLARHDHDTFGIEAIDTIVKAMEDHRDHLVVVLAGY
TQEMETFLRSNPGLRSRFPFIVEFPDYKAADMLDIMIQMANKNGFRIETDAYEGLIELFDQRQIPGRNDSGNGRLVRNLF
EEAVRKQATRLRGLMSGATADTDLQLLTGVDFGIGEKEAFNIENELAGIVGLEKVKLFVRTLEKQLIVDRRRKEAGIHVD
TGQTLNMIFSGNPGTGKTTMARLVAGMLRSMGYLKKGHLVEVDRSDLVAEYVGQTANKTKLVVESALGGVLFIDEAYALA
QDGVQGGGFGKEAIDSLVRLIELHKDNLVVILAGYTEDMQRFVQVNPGLSSRFPLQIEFPDYTAEEMQQIASIMVKARGF
TLAQGVSGLLEPYFNEKQIPGRKDGGNGRLVRNTLEEAIRKQAERLADHPDISADQLNELTVEDFGLSAHVDLAEKRANA
LGALNDVVGLSSVKEFVRSLSAQIEVAKRRQEMGLPKASAQALHMVFKGNPGTGKTTIARILAQRFKELGVIKADTLVET
DRSGLVAGYVGQTALKTKEVIERALGGILFVDEAYALAEGDQFGQEAIDTLVKAMDDYRDRLIVILAGYDEDMERFLNRN
AGLRSRFPNMITFPDYTAEEMLQIARLLVKAQGYVIAREAEASLLSILETYEGDKTAGNGRLVRNLVEKAIRGHALRMSK
KADATAEELSTLMPEDFQGKTGVME
>lcl|NZ_CP026520.1_prot_WP_042231769.1_311 [locus_tag=PC41400_RS01625] [protein=helix-turn-helix domain-containing protein] [protein_id=WP_042231769.1] [location=complement(365983..366354)] [gbkey=CDS]
MSTFCKFETALEILVGKWKPVILLRLFSNGTMRFSELQKAIPDITKKMLTQQLRELEYHDIVHREIYRQIPPKVEYSITE
YGKGMATILQALNDWGVAHVEHLNELYGTNHSVETHKNDAMDK
>lcl|NZ_CP026520.1_prot_WP_042231771.1_312 [locus_tag=PC41400_RS01630] [protein=DoxX family protein] [protein_id=WP_042231771.1] [location=366568..366930] [gbkey=CDS]
MNVVLWIVQGILALGFIYSGWLKAFQYEAAKVSWTWVSDVPKALVFFIGLVELLGVIGLILPQATKIVPVLTPTAAAGLA
AIVLFGAVFHIMRGEYRQIGVNIIFLAFALFVAIGRLKWM
>lcl|NZ_CP026520.1_prot_WP_042231773.1_313 [gene=yeiL] [locus_tag=PC41400_RS01635] [protein=transcriptional regulator YeiL] [protein_id=WP_042231773.1] [location=complement(367191..367862)] [gbkey=CDS]
MEVYKGEKKRLYLEKNSIAPLFSFQVEDFMEVHEYQRDEWIIREGMRPDFLFYVMEGKAKIYVSHQNGKVSLINFINAND
FIGEMELLNDVYYTKGIQALTKTICFALPVHRYCTQLLDDAKFLRELTKFLSVKATFMAAKYSQSLAFPLENRLADFILQ
TADEGIYKEKHVTVCDYLGVSYRHLLHVLTKFCENNYLQKEGRSYRIKQSHVLYELAEALKNK
>lcl|NZ_CP026520.1_prot_WP_042231775.1_314 [locus_tag=PC41400_RS01640] [protein=MFS transporter] [protein_id=WP_042231775.1] [location=367944..369116] [gbkey=CDS]
MNAHRWMSTRFFSFFMTWGIFLPFWSGWMIYTKGITVSQASLIMSFGLIARGLSTLFAFPYLSARLSSKVLLNVAGIGAL
SAILCYIPANSFTGLLTVTILFHLFYPILMPALDSTASVLVQSKQLRHYGRSRQWGSIGFVSIGMILTVFTGRFGDEVIY
WALLLGLIVFVGLSFMRAPDSLTEKRQVDQTKKAGMLHLFRIKHFGLVLLIVILLQAAHAAYYNYGYIYLQHIHAPVYLI
GLIINISVIAEIIFFSIADKSFNKFSVGALLTLSAIGSSVRWILVFVFPNVIMFSLAQTLHAFSFAMGHYAFMKYLIKHI
PQDQIPKAQGIYSALALSWSTAVFTMFGGYLYEIEPRYAFIGMLVCTLPAMLIALVYQKLERRKKALLPS
>lcl|NZ_CP026520.1_prot_WP_042231777.1_315 [locus_tag=PC41400_RS01645] [protein=DUF1697 domain-containing protein] [protein_id=WP_042231777.1] [location=complement(369216..369770)] [gbkey=CDS]
MVYTALLRGINVGGNNKVDMKTLKKSFEQAGMKNVVTYINSGNIIFSTTSRSKNELAQILEKAIMDTFALQIKVLVLHLE
DMKKVISVLPESWMNNDKLRCDVLFLWKAINDESILDKLIFKPEIETVKYAPGAIIWAIDKINVNKGSLTKLVGTELYRQ
MTIRNLNTTRKIYQLMLAAYEDDK
>lcl|NZ_CP026520.1_prot_WP_053228752.1_316 [locus_tag=PC41400_RS01650] [protein=RidA family protein] [protein_id=WP_053228752.1] [location=complement(369777..370196)] [gbkey=CDS]
MSTFTMGHINPEGLFRSEAFSQAITVTGRAKTIYIGGQNAINSKGELIGVGDLELQTKQALHNIKTILAAENASFADLVK
LNIYLVSGSDPLVGYKAFQEMTGSLDNPPLVTVLFVSEFTSRPSCLLEIDGIAVIEGKD
>lcl|NZ_CP026520.1_prot_WP_042231781.1_317 [locus_tag=PC41400_RS01655] [protein=YafY family protein] [protein_id=WP_042231781.1] [location=complement(370282..371253)] [gbkey=CDS]
MAKWDNMLSMIWMLRAGRKLTAAQIADSLEISVRTVYRYIDALCASGVPVVAESGHDGGIRILESFQEAPLFFNSVELKA
LVDAYKFAQGAGYPYTEELESALKKVENGLHEEQRDDLSRRTGGLDVISPARPPSVVPLLRDLEQAVNDGRTVRIDYRKA
NAEQADERDVDPYGLAYDRNEWYAVAFCHRSQAVRTFRVDRIARLEPTEARFEKPERFSASAYFRDQSARGREADGPLTV
IRIEGEPDTLNAVCGHWHMRHYLTERTDREAWFLLDVPTMNKYLPMYLMTFGTAIRIREPLALKRRVQEMAAAIAQHYDI
GTD
>lcl|NZ_CP026520.1_prot_WP_042231782.1_318 [locus_tag=PC41400_RS01660] [protein=crosslink repair DNA glycosylase YcaQ family protein] [protein_id=WP_042231782.1] [location=complement(371374..372567)] [gbkey=CDS]
MIQLTNRQARQFLLLKHGLLGEYKFSGKQGVLDFARQVSCIQYDPIDVCGKNAELVLQSRIKGFTKGMLAELLYEDRSLV
DYPDKNLAIISVEDWPYFERYRQAARQHAKRYPEMEALTTQVRAHIQNHGALSSDDLKLDGDFSWRSAIHWSGGNNSSRS
VLEQMYSTGELIIHHKKGTRKYYDIAKKYIQPNLLNASEPLEDELEHHKWRVLRRIGSVGLLWNRASDAWLNIWGLKAAQ
RNEVFRQLLYEGRIVAVAVEQIKDMLYCLAEDLLLIETVLQNQEPKLRCELIAPLDNLIWDRKLINELFGFDYTWEIYTP
AIKRKFGYYVLPLLYGESFIGRAEIIVERKTGTLVVKNIWYENGVKQTAQLRTALNSCFQKFALFNGCETISAESMN
>lcl|NZ_CP026520.1_prot_WP_042231783.1_319 [locus_tag=PC41400_RS01665] [protein=SMI1/KNR4 family protein] [protein_id=WP_042231783.1] [location=complement(372686..374011)] [gbkey=CDS]
MLFWKKETQLDRIKYKLEKAMRKDTAFLVFGASSHKYKVDEKLTAKELTDWQAKNQVTLPEPYAQFLTKVGNGGAGPYYG
IYSIEKATSYTDSALTTKCVLHPRMTKEEWNHLTKPLINDEDISDLEYDAARDRVMGGMLCIGTQGCEYDMYLVLEGENR
GKIVYTSDFYPDRPFFFVYEDNFLDWYERWLDEIILDYDIGWFGSRMPGDENALIQVYQNAPNEEIKTKALEGMFKFKKT
SQLTIDFLKSVAEQRQNDRTTAIQLICKTSVDAGRDFLLELLHSERNEDFLQALNIVNWHGKSVDLAEFIQVILRSLDRV
HDPETLRHVGYVLESSGAITLQNFAPFLCHNDPNIQTAAIYATRNCTNKSESWETIEQMLMGGDKEVVKNTIIYWGIIPH
EKLLPYYKAEWPEYKNNNNFREKFMGCLKELNLPDDYFDKE
>lcl|NZ_CP026520.1_prot_WP_042231785.1_320 [locus_tag=PC41400_RS01670] [protein=endonuclease V] [protein_id=WP_042231785.1] [location=complement(374031..374540)] [gbkey=CDS]
MIIAVDVYYEENQAKSVGVIFQSWEDSKPLDVIVSYTENPHEYEPGFFYKRELPCIRELLKLTDLNQIHTIVVDGYVYLN
NEKKPGLGHYVYTAFSGKIPVIGVAKSAFHDNEAFVKKIYRGNSSKPLYITSIGMELAVAAEHIQSMYGEYRFPHLLKLL
DKQTKEARL
>lcl|NZ_CP026520.1_prot_WP_042231787.1_321 [locus_tag=PC41400_RS01675] [protein=hypothetical protein] [protein_id=WP_042231787.1] [location=complement(374887..375720)] [gbkey=CDS]
MLKWNSDVWGKLTGPYSSADNVPVLLQQLMQEYSQEVFDELFQEFLFHQNTIYTATYAVMPFLAQLACSTSDEEVRKELF
INCGIIEASRDGRDEAPFPASWAELAEDVGSSVCTELYREYVEAIGKLRVLTKEVFAFTAQHPIDELEKRYILVADAAYR
GSYRPANMLMTFSNGDEYVAVCPACEEDVYIWPNEDHAEILQAYEDDPVFHTGQESQPIVPAASFADEEVRALAERAAAI
GERTLAGHLHYLAGETVCPSCRERISVWPSLLGTFTM
>lcl|NZ_CP026520.1_prot_WP_042231789.1_322 [locus_tag=PC41400_RS01680] [protein=TetR/AcrR family transcriptional regulator] [protein_id=WP_042231789.1] [location=376021..376638] [gbkey=CDS]
MNSQKKLSFIEEARRNQIMACAIEIISTKGYAQASLAQIAKEAGISKGVISYHFSSKDDLIEQIVSDLYESGSTFMTPYL
AAQTTALDWINTYIEKNVLFMKSHRRHVIAVTEIIFHARDEEGKPRHFRNIDTDATINVLVKMLHRGTMEGDFREFTDFS
ARVLAATIRSAIDGLSFQLAASLDLDLDAYAKELVTIFQVAARKV
>lcl|NZ_CP026520.1_prot_WP_241688773.1_323 [locus_tag=PC41400_RS01685] [protein=hypothetical protein] [protein_id=WP_241688773.1] [location=376674..377078] [gbkey=CDS]
MRLIKTFVGGIVLILIGIAWGLLLDEIGMREWLLLLSGIVLGIIAGLVQRWAVARQRLGLITPGKKRLWIIGVIVVLVTV
KVAINVFIPSYLATNNSGIYLSIVYAIGGLLLGHALYLRFKPMPQPAKLRANRT
>lcl|NZ_CP026520.1_prot_WP_042231793.1_324 [locus_tag=PC41400_RS01690] [protein=NAD(P)/FAD-dependent oxidoreductase] [protein_id=WP_042231793.1] [location=complement(377593..378630)] [gbkey=CDS]
MTEQLELYDVTIIGGGPAGMYTAFYSGMRDLKTKLIEAKDELGGRMLIYPEKMIWDVGGVPPILCEKLIAQLSEQAKTFD
PAIVFGQHIIGLEKQADSTYILEAATGEKHWTRTVILAIGRGILRMAKLELEGAERYEVSNLHYTVQELEPFRGKRVLIS
GGGNSAVDWANELVPIAAQVTVVHRRDRFGGHEKNIVRMKESSADVLTPYEVTQLYSGNGKTIERVTITHIESGDSRELE
IDEVIVNHGLKSDFTGIKDWGLDMDDWNIFVSPRLETNLPGIFGAGDFVNYESKLNLIAGAFTDAALALNSAKRYMDPEA
PRMAYVSSHNERFKERNKAIGISDE
>lcl|NZ_CP026520.1_prot_WP_042231795.1_325 [locus_tag=PC41400_RS01695] [protein=helix-turn-helix domain-containing protein] [protein_id=WP_042231795.1] [location=complement(378791..380380)] [gbkey=CDS]
MDIEQRKQAGAAPSVQYVLRSIRLIVAGEQAEMRDLEGSDRNDKESFELLIIMSGTGQGKCYLQGPGRSLSVDQEMMGGS
YYRVCFVTSGYDAASSLLPHKQELACVPFAKTMELLDELYGLRGASDELELFQRFVRFQELLLFLFRQNAPAAEDKGTDI
ERHGVELSIRHIHQYYRELLTVEELASLAGIERWKYTRLFKEATGQVPLQYLNEVRINQAKKWLASADDKLLDIALNAGF
NNEYYFNRRFKQTVGISPGQYRRSRKGQLRVVAPYLEDFIVALDMQPIAQYWHAKWGKQDYLGLNHIPTFDENDGNFDAL
SGYEPDLILLMDRYEQQQYSECRRISNTCILRGQTNNWRTLLRTVADYFGRTELADDAIAKYDFKARSASHKLSQVMKGE
TVAFLRISADHIIQYTEEGQGFVSTVLYGDVGLRSYSAVGVASKANRPGMFNLSVEDLRTLTADHLFITFDKWHSQAEGK
ERELLEQPEWRDLPAVRNNRVYEVDFLTWMNNGIISNGKKIDDILRSLA
>lcl|NZ_CP026520.1_prot_WP_042231797.1_326 [locus_tag=PC41400_RS01700] [protein=ABC transporter substrate-binding protein] [protein_id=WP_042231797.1] [location=complement(380532..381536)] [gbkey=CDS]
MARSGFMKTGWISLITAAMVTVTGCGSTGDQAAATVTTDQPAQATAAPSGAPPQTGTRVVTDGFGEVKIPTKPKRVSALY
REDYLVALGVTPIVQYYNPMWGKQDYLKLEVPLFDVTGSIEALLVSEPDLIIGTGEVDAAQYELYSKVAPTFRLPDDVLA
DSRKTLTLIADLLGISGKAEQVLADYEARIADVKAKLNAAIGDEKLVVLRMNVVDKSINIFGIHNTFVGQILYEDLGLNA
PGFAKAMTEGNIVLSKEVIPKLDADHIILLPSNGTWEEEGNAKALEEMKANPLWKTVPAFKNGHVYPVERSYWQTGAITA
NGLKMDDLLRLLAP
>lcl|NZ_CP026520.1_prot_WP_042231798.1_327 [locus_tag=PC41400_RS01705] [protein=hypothetical protein] [protein_id=WP_042231798.1] [location=381725..382117] [gbkey=CDS]
MTVDRVTGHNENYHSYSYIKQRREAIVNESTIEILHAKADPLKKELADQYSLKRIQHDLELLDEVIQKGFNTLSKMEVIS
PVTAIKAMKMKHKLINGSTGGDTHYGLREIKLREAAREQAITAAILQLSR
>lcl|NZ_CP026520.1_prot_WP_042231800.1_328 [locus_tag=PC41400_RS01710] [protein=XRE family transcriptional regulator] [protein_id=WP_042231800.1] [location=complement(382391..383551)] [gbkey=CDS]
MSVVRVFNGDRLKAARIYRGKTIAELADEVQVSKQSISQYENGKANPSLETLLKLISALGFPRDYFYEKDDKSIKVGTTY
FRALLSTNKKDRLSQEEKTKILAKIYHTLNQFVQFPKINIPPIREGINDIEKVAQATRDYWKLGTDPIPNMVYILEKNGF
IVTSFGTDNANIDAFSQRQEVDGEEYLFIVLGSDKSSAARRQFDAAHELGHIMLHDWNADLELVSREEFRQMENEAHQFA
ASFLLPKEAFLKDLVLPNNLDFYIELKKKWKVSISAMIVRAYQLKAINYNQYQYLMRQISKKGWRTKEPLDDVIQVSKPT
VLRKAVDVIQSNGVLSGDQFMHQLSKNNVSMIKDEVEILLGLDRGTLSEVKQNAVVLSLRHKDVRT
>lcl|NZ_CP026520.1_prot_WP_042231802.1_329 [locus_tag=PC41400_RS01715] [protein=DUF5986 family protein] [protein_id=WP_042231802.1] [location=complement(383579..384340)] [gbkey=CDS]
MDNNTLSIPMADNYKTIIIDAIRNANKKDIMEFRMERKVETFNSVHFLKWDFINTNIIRTLPSDQFQCVKVKRGPLWELV
LIYDKETKYLYAAMQEKRFRQLSERLYKETVHYIDALSSKNNDLEADVDLNPAKQIYLFDTENVDWNEQVEYELKLMLHL
LEGEVTRFVLVTFSTDKGEIASVNAIMPNAELEIAYEENWNEFITADFSTAFELQGTTSEHEEEIFVGLREGVVSHQHDD
LVQLRNDEMEKEG
>lcl|NZ_CP026520.1_prot_WP_241688774.1_330 [locus_tag=PC41400_RS01720] [protein=hypothetical protein] [protein_id=WP_241688774.1] [location=complement(384661..385656)] [gbkey=CDS]
MNKQINTKHQRKQKQWGQENKAPTEILRRESLFKSLGFAAVVSLFVSLLLVGIFSVLILSVKFPRMGWLSFLSEDLKMLA
INGSAVCLGIFLFISGTALSGYLRPPYSIVMSHDRIQFSKWGRRYDVAFEELIGVEQKFRIARFSFFWLQEYILYTPEQV
IKLPIRSIEGRYKVEDRLLQRNEQLNNRKLEALEATAQLQASTDSSISPATRLTLTYSTIKTQRIVVDSEQNRIIFSERN
QLPADQIRKIVVDCNASLENIGGRKITFWQQNGRNFTIKTKNYACSDAEWFTLLQNLHIAARKLNIPLEVDIPPNRPLHI
FSVNDEKERAS
>lcl|NZ_CP026520.1_prot_WP_042231805.1_331 [locus_tag=PC41400_RS01725] [protein=YfbM family protein] [protein_id=WP_042231805.1] [location=385923..386420] [gbkey=CDS]
MGIMACFLSLNDALADEVAQLDNAHIVEKVEELMGKQRCPVYEMDKLWDGLHYLLTGNSASQPIEDHPLSEAIVGVHVLD
TEEFVSAIGSDELPRILKALHSVDRTVLKQHFNPADFRSKQIYPDIWADDEAEERFTELITELNQLTHFYEQSLAQGHDI
LISIY
>lcl|NZ_CP026520.1_prot_WP_042231807.1_332 [locus_tag=PC41400_RS01730] [protein=MerR family transcriptional regulator] [protein_id=WP_042231807.1] [location=complement(386735..387184)] [gbkey=CDS]
MHTVKEAALITGLTEHAVRFYTDKGLVPSVQRNQNNIRMFDEESINWLHGVKCLKQSGMPIEVIKMYVDLCLEGDSTIPQ
RWAIMMEHKEAALVKLEEAKQHVAHLEQKTALYQAILEQRSPDTTNPGNWDKIQHMHCDVFYSPPVQKA
>lcl|NZ_CP026520.1_prot_WP_042231809.1_333 [locus_tag=PC41400_RS01735] [protein=aldo/keto reductase] [protein_id=WP_042231809.1] [location=387358..388209] [gbkey=CDS]
MQTVTLNNGVKMPIIGFGVYQVPDAEECENAVYEALMAGYRLIDTAAGYLNEEAVGRAIKRSGVPREELFITTKLWVQDA
GYESAKLAFAKSLKKLQLDYLDLYLIHQPFGDYYGAWRAMEDLYREGKIRVIGVSNFLPDRLMDLIVHNEIVPAVNQIET
HPFYQQIESAAFMKEQGVQHQSWAPFAEGLNNMFGNEVLDSIADKHNKSVAQVVLRWLVQREVVVIPKSVRKERIIENFD
IFDFELSADDMEQISTLDTRESLFLSYRDPEVAKMMGNWRVDL
>lcl|NZ_CP026520.1_prot_WP_042231811.1_334 [locus_tag=PC41400_RS01740] [protein=dihydrofolate reductase family protein] [protein_id=WP_042231811.1] [location=complement(388570..389139)] [gbkey=CDS]
MRKLVLFMHVSLDGYASDSNGGLGWIPYNEELEKYAEEVVAEVGSPVYGRTTYRMMESYWPTVLENPNASRHDMEHAKWV
QDVKKIVISGSMDKAEWNNTMLIKDNIAEEIKALKEQPGKNLVIFGSPGAAKTLLEFGLIDEFMLTICPVILGGGKSVFD
GGGEKIRLKLLASRTLKSGIIAARYELEK
>lcl|NZ_CP026520.1_prot_WP_042231813.1_335 [locus_tag=PC41400_RS01745] [protein=cupin domain-containing protein] [protein_id=WP_042231813.1] [location=complement(389631..389996)] [gbkey=CDS]
MQYQAINLNEKLSQFNDYWSLKVIGEMNDYQFKLIKIAGDYEWHVHQDTDKVFIALEGEMILDFRDGQVKISKGEMFIVP
RGIEMKPSAEKECHLMLVEPKSVVNTGGPETEITAAHDTWI
>lcl|NZ_CP026520.1_prot_WP_042231815.1_336 [locus_tag=PC41400_RS01750] [protein=AraC family transcriptional regulator] [protein_id=WP_042231815.1] [location=complement(390132..390959)] [gbkey=CDS]
MKIDVNQLAEYLAQTPFQVQGVYRYARNPGVPHADYTDSFPGFVFPLTGKIQFQFNGTPYIFTPGKVVHGGAKMKLSQKV
FGKTDWEYILVLYRICNSELKESSFLHQHFELLTGQSPRLLELIMRLWHVYNQRGGISMFQTEMLFRDVLNEALLCVDNR
QNSCETHTLFERVSNYMQEYYDQSLTIASLAEQNNVNRNRLSYVFRTHAGMGPAEYLLKYRLNMAQRMLFTSDVPVQQIA
QAVGIADPFYFSRVFKKQFGISPTKYREKFINNPC
>lcl|NZ_CP026520.1_prot_WP_042231817.1_337 [locus_tag=PC41400_RS01755] [protein=AraC family transcriptional regulator] [protein_id=WP_042231817.1] [location=complement(391646..393577)] [gbkey=CDS]
MDVLRLIRIWNQVMVRVLDVRLKQAGIDDDLNQYRLPASAFLFAFQGQGELWVNDEVWLCDRFYVLHAGKGSQVAVRSRG
ALNVYIVLYKASLPASASREFHMMMRESDPFLESWGFPPGEPLGMLELLETIRDGWLKSDDGLTRLSAKGDFLRFVHAVL
RERLFGAGELSLSEQVIRYLSRNYRQPISFEELARQLNYSQQYISRKCKEQTGLSPLDYVIRFRMEDAKALLAGTMATQQ
EIASHVGYPDLIYFSRMFKKYTGMTPGQYRKRYGQSVSDYARNTSQSSVVPSRSLRYPFHVSDIDYQSQQDGANKMTRAN
RSFYAAMLLCLTVMLTACGGLANNGGATHSQVAVSASPTAIPQESEASASSGTKIVTTAFGDVEVPAHPKRVAAISYLGT
VLALDVQPIASESFLMSSPYLEGMLNVVTDVGDSFEKLLELNPDLIITHNPRPEIVEKYQKIAPTVSVPYNQFASIQEEM
LYFGDLLDRRKEAERWNEQFEKEIADIRKKVQQTVPEGKTLSIMQEYDKNVSLFGPKSGRGGRLMYEIMGLKPPTAVPEY
MLKESYYKFSLEKLPEYMGDYLILTTEASLKSLQADPIWGSLEPIRNNRVFLWTENQSWYRDPIAVKRQIKELTDWIIET
ASR
>lcl|NZ_CP026520.1_prot_WP_156972840.1_338 [locus_tag=PC41400_RS29630] [protein=hypothetical protein] [protein_id=WP_156972840.1] [location=393706..393843] [gbkey=CDS]
MRSAIKMKEAAPEDIEDDKGWLVLANNFAVTYGTVQLNGGIGEKY
>lcl|NZ_CP026520.1_prot_WP_042231818.1_339 [locus_tag=PC41400_RS01760] [protein=DUF4085 family protein] [protein_id=WP_042231818.1] [location=complement(393970..394839)] [gbkey=CDS]
MKYLTKEWYELCQWTGLHFGMRVHKSAYVRDEALYLRLYKRKEKELVRQRRGIYDVDPRFMLEQDGVQLVALDKWSLDGD
EISEEDTMVYHISSEERDRIQQLIAAYDARPPFDEVKCKEEFSEIQEWGYKEAFHKLPDELSSRIADMRVFALGYCTREV
LRELKQLSKENKKRMMLVMDEYGQAQQAENIPEHIRSQFRFHDCRVTELTSGKQIVMRLDTQGGFTNSNKVTFDAVDEII
RQDEHILGSYWIYNELYRTDCGYEVHILFSGEGMPELIIRCADIVVEQE
>lcl|NZ_CP026520.1_prot_WP_042231821.1_340 [locus_tag=PC41400_RS01765] [protein=hypothetical protein] [protein_id=WP_042231821.1] [location=395041..395421] [gbkey=CDS]
MKSQLRNITIDSQAFVYWYSGGKSFTLNISPKENKNIKITLIFESNPPDEDPLTFWAFYNITAQKNDLETTIHLGKPKHI
AEIISYLMKQRKELFTKGQPHILNNAWDLLMEMGYSNFNPVWVGEW
>lcl|NZ_CP026520.1_prot_WP_042231822.1_341 [locus_tag=PC41400_RS01770] [protein=hypothetical protein] [protein_id=WP_042231822.1] [location=complement(395513..395809)] [gbkey=CDS]
MDNRRFIKAFIKWEKDPSQFPVSEWMRDIVVNYGEIKDKTFYGDTWTVLVRITRHIEGTWDTYADVAFIVDEAPWKLLKT
GYSFNLWAGKDIAVVTII
>lcl|NZ_CP026520.1_prot_WP_042231824.1_342 [locus_tag=PC41400_RS01775] [protein=Ldh family oxidoreductase] [protein_id=WP_042231824.1] [location=396136..397110] [gbkey=CDS]
MRYDLKSAREFAIHVCVAAGASKAVALSLAEATVAAECSGRSSVGFAHLPDYLEGFLCGRIAGTEEPALCFPAPALIQVD
AKGGIAQLGFDRAFEELLTRVSAYGVTVFSLQNSYTAGELGYYVRRLARKGLVALAATNGSALMAAGQSSEAVYGTNPFA
FAAPVDNRAPLVIDQASSATAFVNIRQAAEQGETIPEGWAIDEKGHNTTDSREAIKGALLAFGGARGANIALMVEILAAG
TTGANWSLDAPSFDDGSHSPGVGLFIVALNPQLLVPDFSARMASQIERLSSKGIHIPGDSYTEDKIEIDLSSSLVATLEK
YCSH
>lcl|NZ_CP026520.1_prot_WP_255246856.1_343 [locus_tag=PC41400_RS30520] [protein=hypothetical protein] [protein_id=WP_255246856.1] [location=complement(397257..397382)] [gbkey=CDS]
MQQLAEGNRSGAARMHIVTKAKANAVVYEKPVSHPDEVAGM
>lcl|NZ_CP026520.1_prot_WP_042231827.1_344 [locus_tag=PC41400_RS01780] [protein=DUF4261 domain-containing protein] [protein_id=WP_042231827.1] [location=complement(397508..398383)] [gbkey=CDS]
MESERVLQQNLDEKSEFQRLFQMYLLFEEKAERPNAEAVRNALVTRCGKTDTVSAADALSSFAVEAYKVAYQEGELPAQV
VMADVTPFQPETITDLERTQFWTMPDAEEVLQKCRYKLLISDFMAAGLDYKSRSALLADWLETAVSLFPTCKAIWIPSSG
KLLHTAEIVDNPYEGAARFLQFGMNIRYFTIHGTEDSLIDSLGLFAIGLPDVQYHFHTLDPNDVSRHAFNVAAYLFEADV
PVSDGETIAGLLNGEMAPDAHWPCRFEMALIQPSREVMDVSPGEYVAGDRE
>lcl|NZ_CP026520.1_prot_WP_042231829.1_345 [locus_tag=PC41400_RS01785] [protein=hypothetical protein] [protein_id=WP_042231829.1] [location=complement(398620..398973)] [gbkey=CDS]
MGVSVYYTCMRNHNLTNSEEQEITAIIDKYNAGFELKDIGETFCVYDYDRDEPTVIFAGSTKLPSSDDFEDTLHALFYWL
TCLTDIRRSISNGDWHVHLDDTDAIWDEETGWQMPED
>lcl|NZ_CP026520.1_prot_WP_129112442.1_346 [locus_tag=PC41400_RS01790] [protein=hypothetical protein] [protein_id=WP_129112442.1] [location=complement(399175..399939)] [gbkey=CDS]
MMMNNLELHVKLEQLIKEKYPKHFRKEVFNLENKIYNMISNQLEANSVCFFNDDKCSNNSKPIGSHLLQQNGVLSKIAIN
NHVIMFRRNFSEPYKIKFKPVSIESASKCPVFCLHHDTSVFAPIENSKQNCSMEEYIFLITLRSFADEYTKRHEDFKRMK
VLYNNINSKEFEDYCKLTGASTEEFKSYIESNRNELEDMLEQYNFLRLKLLTSYKNQDYECITHKQFQSNKIGIAFSTTM
IIPISEEQEFLDTN
>lcl|NZ_CP026520.1_prot_WP_042231836.1_347 [locus_tag=PC41400_RS01795] [protein=hypothetical protein] [protein_id=WP_042231836.1] [location=400436..401326] [gbkey=CDS]
MRDAAEGQQKHGQQEHFETLPLFSTTDKNGRMTMLRPGHRVGRAAPLMPWLLTAAALWSLTGSVPFGALLGMAPTPAISM
LLGHPVTVGVAVLLLFVAIGTTGGVYSRSIEQFGQTRVAGLFATLSIAGGLAAVAGILLFWTLTSDVSRPFDLEAIATSP
TVPPELGAVVGASFALWAAIAFLRLPGSIAHARRRQADIERLREEGLSCTGTLTALNFTNSWLFNFPIFTVEVNYIVDGA
PRVVSAHMRTSADRVPVVGSILIVLTDNRGTTHVELDLESGATFEPDVGKYAPSDG
>lcl|NZ_CP026520.1_prot_WP_042231838.1_348 [locus_tag=PC41400_RS01800] [protein=SMI1/KNR4 family protein] [protein_id=WP_042231838.1] [location=complement(401672..402511)] [gbkey=CDS]
MEDVKLTGTYTVDNCGKKEFYGTYYSSEVLSEGKGDKAPKLVQNHGASDEEIRNMEQWLKTLSYIVHGSDISLGCTEEQI
EDAENRLGVSLPAELRIYYCTVGNDPLLTTDRNSKKKDRYLPLDEIHVSDGNIVYRMRKKEPFALSAVKRQMMFRDDEAW
YWERNMESYCENVMITAAIFAISHMKQSAKGRLKGELVTSLQARKLAEQQFSDTFRALDTYYHPFCALFYNERERRLGWF
RSNGMMADILIGTQTAKEIEDFGEMHGNIKFKYSEVSHY
>lcl|NZ_CP026520.1_prot_WP_042231842.1_349 [locus_tag=PC41400_RS01805] [protein=hypothetical protein] [protein_id=WP_042231842.1] [location=complement(402765..403307)] [gbkey=CDS]
MRMPSDSELTLLFDGDGRQLQQFIEKHHEKLKRYQLMDDENELADELAIKEQIPIVFAFVYSVTVDWGEDDEEIVRLFGK
RLPDETVEVTSTDKGLDVTYNGQLHSIALTFTGNDRYITIRGFQELIRDKYEIRLFEASYLSDTHDFLILPKRQWEELDD
RYPVRNREIFRVIDDELDFP
>lcl|NZ_CP026520.1_prot_WP_241688775.1_350 [locus_tag=PC41400_RS01810] [protein=ATP-binding cassette domain-containing protein] [protein_id=WP_241688775.1] [location=complement(403819..404610)] [gbkey=CDS]
MSVIRLENVTKKYEGSMIFRDIYFRVMQGERIGLIGRNGAGKSTVFKLIMGKEEPTSGRVEINPQLKIGYFSQFSELSGS
LSVQQELEMCFEQVALIERELLEVGDKLGLVTDNNEMERLLARQAELFEQMDHLDGWNVSVEINTVLTKLGFNDRSRNQP
IDELSGGWRNRAALAKLLIELPDVVLLDEPTNYLDMEGIAWLEQWLYRFKGAMILVSHDRQFIDKVVTRTIEIENYHFQE
YEGNYTDYDCVTTNEIIKQLFIN
>lcl|NZ_CP026520.1_prot_WP_042231844.1_351 [locus_tag=PC41400_RS01815] [protein=serine protease] [protein_id=WP_042231844.1] [location=complement(404656..405516)] [gbkey=CDS]
MENNPSRNKENFLSHSEEKQEHENDHPYADEEWDFESEEEDDDGGSSANRRKWIRNTVIFLLVAALVGNILAFWPQIYNM
KTLPLLFGSRELSSQAEIQSYKEAVVTVSTDNGKGTGFHINGGYIVTNYHVIEDNGYIIVKFPGQSQSYKAELSGSDPDL
DIAILKADIGDQRLPFIEVERDNKQWEPDEQIYVIGNPLHFTQIAIKGAIIGLVPIQGRQTPVMALDAPVYNGNSGSPVI
NKRGKAIAVVFATADVEHQDQMIEAGLAVPIADMETLLRTSLPPVE
>lcl|NZ_CP026520.1_prot_WP_129112443.1_352 [locus_tag=PC41400_RS01820] [protein=IS3 family transposase] [protein_id=WP_129112443.1] [location=405759..407090] [gbkey=CDS]
MRKIRTEFTETETRLLESNPNVSRVSGRNISYAPAFKLAAVQANQAGEPPMEIFLKAGFSIELIGQRTPTESLYRWRETY
AKYGEVGLLKERRGTGSTGRRSKTESSAEMKLKQAEARIKLLEAENELLKKLEALERRNSKGLTPSERFQLINQTIRKHE
LRRLTRYLCQLAEVSASGYYRWCSAEESRQVRETADRYGFELIKEHFEALNRKAGALVIKMRLEKCNGVVMNHKKIRRIM
RKFGLVATIRQANPYRKMAKATQEHRTCPNLLERQFDQGEPEKVLLTDITYMRYGSGQWAYLSCVKDGATKQILAHYLSS
KLELSLVERTMTRLLKRLDGNIHPDAIFHSDQGMHYTHPKTRLLIAEAGFKHSMSRKGNCWDNASMETFFGHMKDELTYK
DCKSLKELRLRVNEYMTYYNSQRYQWTLKKMTPDEFRSHLLAA
>lcl|NZ_CP026520.1_prot_WP_042235582.1_353 [locus_tag=PC41400_RS01825] [protein=ATP-binding cassette domain-containing protein] [protein_id=WP_042235582.1] [location=complement(407235..408974)] [gbkey=CDS]
MIKVDNLSFSFPQKELFNNISFTLEEAQHCAFIGTSGSGKSTLIDILMDPERYLFDGKLEMDPDCTIGYVSQFSELDNTK
ETTVFEYIGEPFIKLQNEIQSICTEMETSTDIEPLLEKYQFALDALDAIGGDDFESNIHKQLNVANLMKRRDSKVADLSG
GEFKLIQVMKEMLNRPDILIMDEPDVFLDFENLNALKKLIHSYKGILLVVTHNRYLLNHCFNKIIHLENREIQEFDGRYI
EYNFSLLQTKMELQELAVAEQEEMERNDTIINNLRVIATYNSDASRGRALKARVRFQERLEARRIKAPFVEIKQPNISFG
VDNEMEDAIVVSVHHYSVGFDELLLDNVNFEIKSTDKVAIIGANGTGKTTLLRDIFKNNHASIEINADVKVAYLSQLQGE
MLKDANTILEEFIDAGFKTYDEIGLYLFNYGFEGETLSQKIESLSGGEKNLLQLAKVSASKANVLLLDEPTSHLDTYTQL
ALEKAIEKFKGAILMVSHDFYSVVNGMDYVLIIDDKTIRKMTLPKFRQMIYDSHFDKDYLETEQKKKSVEMKIELALKNT
DFELAKGLVEELEELIKWL
>lcl|NZ_CP026520.1_prot_WP_042235717.1_354 [locus_tag=PC41400_RS01830] [protein=hypothetical protein] [protein_id=WP_042235717.1] [location=409268..410452] [gbkey=CDS]
MTITESNRLSELSIEECQGLIAGCAEGAASSPFQINYGNEVRSKVRLFLSELTPEEAWLLELNIGLFLLRKQSTDRHSGY
FAQVAASETIDTIEAHLNSSLPPEIAVKHQGRLLETAVYIRETVKANAWFPPAYLDIYVELWLILVSSDADSPRLFKEEL
SHLADGFGKENKLFPLVARAWIHFWLQEDQAAWRLLEAAERHRLNPGHVFRFLRVLEEAGEWIRLEAWLSHCATERVGRS
PGSLDTYGRYWEAVVGQLPEAERNMWSAITSLLPYSSSLYEESLMRFGRWRQWVDYQLSLGSDPLDFRAKDLQPIEKEAP
EALLPFYHQGTEKYVLLRNRHGYKRAVKLLKRLAKLYKKLKREQRWDVYIETFVSRHSRLRALQEELRKGKLIP
>lcl|NZ_CP026520.1_prot_WP_042235584.1_355 [locus_tag=PC41400_RS01835] [protein=DEAD/DEAH box helicase] [protein_id=WP_042235584.1] [location=410449..413442] [gbkey=CDS]
MTTTIDNLDTIAVRMVLTSHGDALFFADPDSLREWTGLQLKHRLFAWHEPSFYGTELELVKVGELEAIVLPAEEVIPFFA
SGPLLAHMEWKWEDDAARLAELAPLLAECLEKRLYAPDLAAYRSGSLRWSWDAAAVGATFGQARRDELDLKNSSWSDEAR
DGLKAAFSAAVMSRFYGTEPDEADLRREFPALFARENAAAATAGLDDESWLVQIGWKADVAPFRPLLQLLEPCEGDEHSR
WRLRFVLQDKSDPAKLRRVRLGDDGSAVGKWPDAWAEAIRSRSDGWLQRLRASLPHDRLSRGEDVLGKPLDDEAAWRFLN
EDSRRLLEAGWQVLLPAWWEAASRKKPRLRAAVQSEDESKERGGIKSLFGLEAILSFNWRISIGGADLNEEEFDELLARG
ERLVKFRDRWIVLDPALIAQIRRMMEGMDKSQGLSFQDVLQLHLLSQAETDAGADGGATDAGAETFGTDAAHVELDVELN
AHLTGLFAKINQQSEWPRLDPPAGLRAELRSYQQDGFTWLAFLRRFGLGACLADDMGLGKTVQLIAYLLYAKEQADEGMG
LPSLIVCPTSVLGNWQKEISRFAPSLRVAMHYGSGRKNGDAFRNEARSVDVILTSFATASLDQETLAGFQWGAVCLDEAQ
NIKNAGTRQAAAVKSFPALHRIALTGTPIENRLAELWSIYDFIVPTYLGAAKSFQERFASPIEREQDGRRTAELKRLVKP
FMLRRKKKDPAIQLDLPDKNEMKTYVPLTSEQAALYDNCVKELLEKLKKLDGIQRKGAILGALTRLKQACDHPALLDEDM
GTELEDGPLHTEAIVARSSKLERLLAMVKELREADERCLIFTQYIGMGKMIQDVLQRELDEPVLYLNGSTPKAQRDRMVE
RFQSRDLPPSEQPNVFILSLKAGGVGLNLTAANHVFHFDRWWNPAVENQATDRAYRMGQTRDVQVHKFISLGTLEERIDE
MLVNKQQLSDNVISSSEGWITELSTDALRELFSLRRD
>lcl|NZ_CP026520.1_prot_PC41400_RS30155_356 [locus_tag=PC41400_RS30155] [protein=DNA alkylation repair protein] [pseudo=true] [partial=5'] [location=complement(413645..>413743)] [gbkey=CDS]
TGLYKDEMVSKGCIPNYLPEFIAIEVNKRNNN
>lcl|NZ_CP026520.1_prot_WP_042235588.1_357 [locus_tag=PC41400_RS01845] [protein=hypothetical protein] [protein_id=WP_042235588.1] [location=complement(414618..414998)] [gbkey=CDS]
MIDMQYINGLETEMDIMKLVDNYNEIKSILQQYPQEQWSYEMILAYARALNYLQDYSEQTNPFEQAYGLLQTVSDKGNTD
AEWHLIMASVSFNLQLYAEAVKHAEQANTIIAGCADDMLALYSQYD
>lcl|NZ_CP026520.1_prot_WP_042235590.1_358 [locus_tag=PC41400_RS01850] [protein=hypothetical protein] [protein_id=WP_042235590.1] [location=complement(415449..415832)] [gbkey=CDS]
MNLMELRIPQGFAIYYNRFYDVEPISDPNEDDFLTNWTFFTQDLLQISKMELEKGSWNVPKDENRRIHILLGWHPDSRAS
GEYGLAITNGKWDTLMEISSRDRFEIKQTLEKWLEKLNEDQNYLKNG
>lcl|NZ_CP026520.1_prot_WP_042228745.1_359 [locus_tag=PC41400_RS01855] [protein=hypothetical protein] [protein_id=WP_042228745.1] [location=complement(416578..417096)] [gbkey=CDS]
MNQDSKIEVRKSILYNSLNELLRKKMPAICRDIEKAISDRNYTDVMNNIVCLLETHIWGQDTQWRVEAEVECYPCHESGR
PFAIGGFVFLEEVAGQKIIHFKTQMYNSFDESTIKKKDTIKINYDKVNDYEFIQPFIQETEEFLDQFYNVFMTGSDIFVR
EWRLAKSEQENL
>lcl|NZ_CP026520.1_prot_WP_129112444.1_360 [locus_tag=PC41400_RS01860] [protein=hypothetical protein] [protein_id=WP_129112444.1] [location=complement(417553..418107)] [gbkey=CDS]
MGFELSVQLKGLNDKIIPAWLAEIRKFNMDVDIYPNFSFNNHSGFLPFKIVTYDCPNRSLNNIELLTGYELFVNRIEESI
KRQSLFTKLFSGQKQLSPIEKKLSEADTEVIFKISARDSFEFRIGWYSAASLALLCDGVLTDLYEGVQLEGQEMIRRAHE
VVLEDERLIKENEWRVHEFKEWLG
>lcl|NZ_CP026520.1_prot_WP_042235599.1_361 [locus_tag=PC41400_RS01865] [protein=DUF1963 domain-containing protein] [protein_id=WP_042235599.1] [location=418537..419220] [gbkey=CDS]
MKKYKIDSFTRASEPIRTPITKFGGQPVWLTEPQWPLSSGWDKRPMMFIGQIELNPQLFPTSKVPKPLMAYLFVTHAQNM
EEGFFDPDIIDPDGGENAIIIQPGGEPLIDTAPLANGPTLWDDNGESYEGIPQLQEAQDPEFLDSDSYILLSKDEQKVYS
DEVDGDKIGGVPYFFQGDGWPDDSNWRLLLQLHANFKPFCLLVGASPTIFAFVNEDLTTGKLVIQDC
>lcl|NZ_CP026520.1_prot_WP_009673450.1_362 [gene=rlmH] [locus_tag=PC41400_RS01870] [protein=23S rRNA (pseudouridine(1915)-N(3))-methyltransferase RlmH] [protein_id=WP_009673450.1] [location=complement(419388..419867)] [gbkey=CDS]
MHITIASVGKLKEKYLVNGIDEYVKRLGPYAKVQLVEVPDEKAPESMSPAEEQQVRAKEGERLLAKLAPDAYVVALAIDG
EMWTSEQLAGSLDKLATYGRSQVAFVIGGSLGLSSEVLRRADAKLSFGRMTLPHQLMRLVLVEQVYRAFKINRGEPYHK
>lcl|NZ_CP026520.1_prot_WP_042235719.1_363 [locus_tag=PC41400_RS01875] [protein=gluconokinase] [protein_id=WP_042235719.1] [location=420271..421797] [gbkey=CDS]
MGLDIGTTSTKTIITNEKGTLLGAGEAGYPLAVPQPSWAEQDSDLMFEAVLQSIKEAIQRAGIDGKQLEAIGFSTAMHSV
IPMDADGRPLMNSIIWADNRSVAQVQRLKDENLSSMLYHKTGIPIHPMTPLPKIMWLKEERPDIFEKAVKYISIKEYVLY
KFFGRYVVDHSVAGGTCLFNLNTLSWDPDALRIAGIREDQLSELVPTTHIMRGMNREFADLLGLDPEVAFVIGAADGALA
NLGVGAVEPGVVALTIGTSGAVRTVVPKPMTDSKERTFCYALTENHWVIGGATSNGGLMLRWFRDEFSWPEVEEARRTGR
DPYDVMIAAAKEVAPGSEGLLFLPFLSGERAPYWNADARGSFFGIGLHHKRPHFIRAVLEGILFAVYGIGIALRDLAGPA
TEIRASGGFARSPEWRQIMSDMFGYPVHIPKSHESSSFGAVVLAMYAVGALERLEDVNKLIPITERQEPNIEISPTYIEL
YYLYDRVYRKLEQEFSIIAEYQRRTTTK
>lcl|NZ_CP026520.1_prot_WP_042235601.1_364 [locus_tag=PC41400_RS01880] [protein=DUF4878 domain-containing protein] [protein_id=WP_042235601.1] [location=complement(421880..422287)] [gbkey=CDS]
MKPFLETYKTIMVPSTAFFIFVLLGMGILFFKQDASTEAKRSANRYLTAVIKHNAEVVEENVVYDLTLTPKERSDFKNYI
WANDIKDFEIKDALLSEPGRYEMRVLIMKEASQEESRITVVKVDSKWKVLIKKPV
>lcl|NZ_CP026520.1_prot_WP_084706695.1_365 [locus_tag=PC41400_RS01885] [protein=CxxH/CxxC protein] [protein_id=WP_084706695.1] [location=complement(422452..422619)] [gbkey=CDS]
MYSVCKEHLELAIDVFVDEYEDAPDIVDLGKVKFAEWEPPAHCEHCKEKPQYLVV
>lcl|NZ_CP026520.1_prot_WP_042235603.1_366 [locus_tag=PC41400_RS01890] [protein=trypsin-like peptidase domain-containing protein] [protein_id=WP_042235603.1] [location=complement(422639..423877)] [gbkey=CDS]
MSLFDDDFYSTKQRKDRRKQEEGGWSKGKPFSSANSPLTGLRGLTDRGSRTVWLAGGTGAVIMLLVVLLVQALAGTGSAK
PVALSPKQMEDSYNNAVVQAAERVKLAIVSIVNLQPNDKNELKQTGIGSGIVFEKNGSKVRIVTNYHVVDGGKQFEVYTA
SGEKKKAELLGKDQITDLAVLETDASNLTSIAEFGDSGSLKAGETVIALGNPLGLGFSPTVTKGIVSSPKRSIPISFAMD
GQYDWEMELIQTDASINQGNSGGALVNLNGQVVGINSLKISDMGVEGLGFAIPIKDAEPVIRSLIDNHKVIRPLMGVSTQ
ELQLFQGTEELKLPADVKRGLIVIEASGPAKIAGLKSEDVIVAINGKKIETTVQLRKFLYNESKIGDKIEVAYYREGKKQ
KTELTLAEAGEN
>lcl|NZ_CP026520.1_prot_WP_042235605.1_367 [locus_tag=PC41400_RS01895] [protein=hypothetical protein] [protein_id=WP_042235605.1] [location=424051..424263] [gbkey=CDS]
MNNRIRTHLDEVQLIGKLADLKEDHYRNSLILTALIDVLIDKGVITTQELQNRMDKLDNSFSPYPAPPIS
>lcl|NZ_CP026520.1_prot_WP_042235608.1_368 [locus_tag=PC41400_RS01900] [protein=MBL fold metallo-hydrolase] [protein_id=WP_042235608.1] [location=complement(424232..425038)] [gbkey=CDS]
MGLRFTVLSSGSTGNATVICDGETKLLVDAGLSARRVEQLMKEREVSGDQIDAILVTHEHSDHIKGLGAIARKYDLPVYA
NTKTWEALDKHIGTIAEPNRRIMETGTEEYFGNIRVESYGISHDAAEPVGYCFHSGGQKLAMATDLGYMSTKVKDVIGDS
DAIILEANHDIEMLRMGRYPWNIKRRILSDLGHLSNEAAGEALLDIMTARTRAVYMAHLSQNHNLMDLARLTLNSVLEDR
GVSLKERNIRLMDTYFDRPTKWEELGTD
>lcl|NZ_CP026520.1_prot_WP_042235611.1_369 [gene=yycI] [locus_tag=PC41400_RS01905] [protein=two-component system regulatory protein YycI] [protein_id=WP_042235611.1] [location=complement(425084..425848)] [gbkey=CDS]
MNWGRAKTILILSFFFLNVVLGYLVWTGKAKQTELATDMTGVLEETNKLLKGKSIQLSHELPKEAPKLKEITFKFVESYQ
PNVILKLQYPFKIASLLNRISPKDDLPKTDIPKLDAYKHDPILSKKGMYVLNQMYNDYPMFDVQLKLYETDAQITSYRQA
YVEVESGGEDKEQKVIPAYIAVRSLVEKFLLDGSVITDVRLGYHGQLFDSQTQYMVPNWRVTINNGDIYYVHAFNGAVEM
QTASTEDAAADTKK
>lcl|NZ_CP026520.1_prot_WP_042235614.1_370 [gene=yycH] [locus_tag=PC41400_RS01910] [protein=two-component system activity regulator YycH] [protein_id=WP_042235614.1] [location=complement(425950..427236)] [gbkey=CDS]
MIEKLKSVTLTVLILLSLLLSYLLAYSTPNYEPLPPSEYISSELEGTKAKLEDLLFPDQIVLHFGNQQHTVLYPDSTSYS
TIFESVKQRKMERFRKVILNQERTNWDEIREKQEGVEVRFRDGLPLSVLQNTFQIKGDIPDDTELITRIWIFASSDKKSA
GTYLFTDSSNTAFEVLGADFTAKDVENFTAIAGTNPPYVAASGGEYYLPVNPLPAASYDFGYKEFPTEKLKSSLFADPSL
TRNLKDRDGSQIYTDAKRGLQINQEKKWMKYTDPLAVTDSKNSIKENLDAGVQFINQHGGWNGTYTISKPQQQHAFSNQM
FVFRQVYDAFPILTEQKEGYGAIHITLQKGVVSNYERSVVYSDLKLINQQKSELIGGEALNAKITNHPKRYSIVNVFPAY
RPTLGVKTVNLSPVWAIEFKDGTYEFFR
>lcl|NZ_CP026520.1_prot_WP_042235617.1_371 [gene=walK] [locus_tag=PC41400_RS01915] [protein=cell wall metabolism sensor histidine kinase WalK] [protein_id=WP_042235617.1] [location=complement(427233..429065)] [gbkey=CDS]
MKVIRFFQSIQAKLIIIYVLLILIAMQLIAAYFYQTLDSHYKNEFRDSINSQATLIAGLVESQYLNEPKNSDTKNLSEEL
TYFVNSLFAPGNTEIQIIDAGGVVLSTSLEANKQIVNQKTTKTEVTRALQGIRDNDRMFTDVDNTRKFATAKPIGSGVRV
LGAVYIVSSMEDLYKTMSSINRILIVATLIALAFTAGLGVILSGTITKPIKEITQQATAAAEGNFDQSVKVYGKDEISQL
GHTFNFMMNRLKEAIFINEEEKEKLASILTNMNDGVVATDEAGQIILMNRRAMQILQVKEEQSAQRHISELLGLTREEID
QYVNGERNTTLIDVPLPDDEDVQTVRVTFTTIHRRGDGVTGAIAVLQDVTEQEKLEQSRREFVANVSHELRTPLTTIKSY
LEALEDGAMEDKQLAPRFLSVTRNETERMIRLVTDLLHLSRLDSKQALLTKEQTDIADMLEEVADRFSFQLRQKSIRIRI
VAEPGIRPVALDRDKIDQVLDNLVSNAIKYTPDEGAITIYARMAEGGQLEISVQDTGIGIPKKDQPRIFDRFYRVDKARS
RNMGGTGLGLSIAREIVKAHGGAMSLESEPGSGTKVTFMLPVLPEEENAS
>lcl|NZ_CP026520.1_prot_WP_009673491.1_372 [gene=yycF] [locus_tag=PC41400_RS01920] [protein=response regulator YycF] [protein_id=WP_009673491.1] [location=complement(429067..429801)] [gbkey=CDS]
MFGTNGKILVVDDEQPIADILKFNLEKEGYQVVCAYDGGAAVELAFSERPDLILLDLMLPVKDGMDVCREIRARLNTPII
MLTAKDTELDKVLGLEMGADDYVTKPFGNRELLARVKAHLRRQSKTTPLGQAEPEPEEANGIRLHGLFIDSDMYVVYRDG
EPLDLTHREFELIHYMAKHPGKVMTREHLLQAVWGYDYYGDVRTVDVTIRRLREKLEADPSKPEYIVTRRGLGYMMRGTK
NGGF
>lcl|NZ_CP026520.1_prot_WP_042235620.1_373 [locus_tag=PC41400_RS01925] [protein=M23 family metallopeptidase] [protein_id=WP_042235620.1] [location=complement(429947..431461)] [gbkey=CDS]
MKGFKAVDWARKLWNQSRAQWHSLTGNKAEPSTELAAGPAGRGKGFFLYSVSSIAIVGFLTVSGQQYVKANLADIYPVKV
GQETLGYISSKEVLQEYKQEREEKLGKQYPDVHMALPDIEEPKPEKAFMKKTDDGKVLALLDEKLKPIASGVELVIEGKT
YAIVKDQATADRVLAEIKKPFEQKEKESKVAVLSADGSPSAAPTTVEKVEFVQQVKTEEAVDIQPQDVMKPEELIAKIKT
GGVQPTKYTVVAGDCVGCIAQKLNIPKQVIYDKNPWIHDDMIREGDVLDLTVIQPDLSVKTVEKMVENEDVHFETEVETD
PTMKEGIVETISPGKNGLKQVTYELTKINGQLTDEKIVGEDIVEQPVTEKVRKGTKVVKGEGTGTFAWPVLSPSITSTFG
TRWGKMHKGIDIVGNHTIMAADNGKVVSAGYKSDYGNYVLIDHMNGYETMYAHMSKVETTAGTIVEKGEQIGVMGSTGDS
TGVHLHFEIHKGGGLENPLKYLNR
>lcl|NZ_CP026520.1_prot_WP_042235624.1_374 [locus_tag=PC41400_RS01930] [protein=hypothetical protein] [protein_id=WP_042235624.1] [location=complement(431669..432394)] [gbkey=CDS]
MKWLGRLGRWTAGTVVMSFLSIFTTYVIVTTMVDEVLKQFQLPPIGNKLSFSQITGKLGEQFGIGAVGKGGSVNDRVERI
AASSSKPSPSPSPTPTSTPGGAEKPVTGSLDGSGMPSPSSSPVQGGVSSPEPSGMPDAVAVMGELKEGQASRDPKDAQKR
DVVMSTDDFSKKKDALSSGDKQKIFSILVAKLPAEEVQKLSAMVEDGITAEELKEVDSKLKKYLNSTEYEQLSGILKKYG
G
>lcl|NZ_CP026520.1_prot_WP_042235627.1_375 [locus_tag=PC41400_RS01935] [protein=adenylosuccinate synthase] [protein_id=WP_042235627.1] [location=complement(432687..433973)] [gbkey=CDS]
MSTVVVVGTQWGDEGKGKITDYLAESAEVVARYQGGNNAGHTILIGGKKYKLTMIPSGIFNHDKMCVIGNGMVINPAALV
DELQYIRDNGFSVDNLKISDRAHVIMPYHLLLDELEEDRKGANKIGTTRKGIGPCYMDKAARNGIRIADLMDAQEFERKL
RHIVEDKNTLLKQVYGRDGLDADAILKEYLGYAEQLRMFVTDTSVVLNDAIDTGKRVLFEGAQGVMLDIDQGTYPFVTSS
NPTAGGVCIGSGVGPTKINQIIGVAKAYTTRVGDGPFPTELNDAIGDQIRETGHEYGTVTGRPRRVGWFDSVVVRHARRV
SGITGLSLNSIDVLTGLETVKICTGYKYKGEIIEHYPASLNMLSECEAVYEELPGWSEDVSNAKSLSDLPENARHYLERV
SQLTGIPLAIFSVGRNREQTNLVRPVYS
>lcl|NZ_CP026520.1_prot_WP_009673460.1_376 [gene=dnaB] [locus_tag=PC41400_RS01940] [protein=replicative DNA helicase] [protein_id=WP_009673460.1] [location=complement(434123..435478)] [gbkey=CDS]
MNEEMFMGRVLPQNIEAEQAVLGAILLDSDSLITAMERMSPDDFYRPAHQHIFEAMIELAEEDEPVDLVTLTARLQNKQQ
LEEIGGVAYLSELANAVPTAANIDYYGQIVEEKSLLRRLIRTASQIVSDGYASADDVGMMLNDAEARILEISQRRASNGF
ISIRDVLMEVFERVEFLYSHKGGSTGIPSGFNDLDKMTSGFQRSDLIIVAARPSVGKTAFALNIAQNVGVRAKETVAIFS
LEMGAAQLVQRMICAEANVDATRLRSGFLEGDDWEKLTMAIGSLSEADIYIDDSPSVTVADIRAKCRRLKQEKGLGMILI
DYLQLIQGRGKGDNRQQEVSEISRTLKQIARELNVPVIALSQLSRGVEQRQDKRPMMSDLRESGSIEQDADIVAFLYRDD
YYDKESEKKNIIEIIIAKQRNGPVGTVELAFLKNFNKFVNMDRAHQEMQTG
>lcl|NZ_CP026520.1_prot_WP_042235630.1_377 [gene=rplI] [locus_tag=PC41400_RS01945] [protein=50S ribosomal protein L9] [protein_id=WP_042235630.1] [location=complement(435544..435987)] [gbkey=CDS]
MKVVFLKDVKGQGKKGEIKEVSEGYATNFLIPRGLAAAGTDGAVKQVDQQKKAEQKRKDQEKEDAEVLAGKMNETTVTIK
TKAGEGGRLFGAITSKQIAEQLEKAGYKIDKRKILLNDPIRSLGVTKVPVKLHTEVTATISVQVSEE
>lcl|NZ_CP026520.1_prot_WP_042235632.1_378 [locus_tag=PC41400_RS01950] [protein=DHH family phosphoesterase] [protein_id=WP_042235632.1] [location=complement(435984..437951)] [gbkey=CDS]
MPKFLSKRWHGQHMVWTFALLGALTIVLFLYEWVLGVAALLASGLVLYFSWKAESAFRHDLNAYLGTITHRVKKSAGDVI
SELPIGMILYNEEKEIEWHNPFVAKLAKKDSVIGDSLEDVLPSWKPGKEKEKEETYEVEQDGQVYKIWHRAEERLLYIRD
VTDYVRLVQKHDEEKIAIGIVLMDNLDEVTQGMDDQTRSIMLARVTGEITEWAHRYGVYLRRTASDRFLILMNQKALREL
EQNRFDILDEVRDLTMEYKLPLTVSVGIASGIEELEELGRVAQTSLDMSLGRGGDQVTVKAGQRISFYGGRTNAVEKRTR
VRARVISHALRDLIKESDKIIIMGHRFPDMDSIGAAIGLLKAVQISNKEGYIVLEGINPSIEKLMAMIEEDEKLNRWFIT
PAQAMQITTQRTLAVVLDTHRASMVAEPKLFNLTHRVMVVDHHRRSEEFFHDAVLVYMEPYASSTCELVTELLQYFNEKM
SLDALEATVLLAGIVVDTKQFSLRTGSRTFEAASFLRRNGADPTLIQKMLKEDLQAYIEKAEIMRHTEMMYDHIALAVGE
QNHKYSQLMIAQVADTLLNMTGVFASFVICERPDGLTGISARSLGAMNVQIVMERMGGGGHLTNAATQLELTPREAAAKL
KQILQEIEEEEGLFE
>lcl|NZ_CP026520.1_prot_WP_042235637.1_379 [locus_tag=PC41400_RS01955] [protein=DUF2232 domain-containing protein] [protein_id=WP_042235637.1] [location=complement(437969..438892)] [gbkey=CDS]
MTKKNLQFIGWGLLYILFLLSFMTSYLVITASFLMIPVLILFVKLDAKRFTAFYLGILLIVNLLLGWTGFVVLAISLFLL
PPVIVIGTLYKKKAPAHLVITYGTLTFLAEMLLTLVIGFLFKLNFIKKFKTIMLDYLNNMQPAVKGMLPTQGDMQWYVDV
LVQIIPLDMILFSLYFIGVTHLIGRRLLNRMGEDIPGMPPVREWKLPRAFVWIYLIAFAADMLFVSATNSLMSALLINLL
PLLMFAFAIQAISFLAFVAHYKGWGKLLPVAGTILLIVFMPFLLVLYSLLGVFDVAFPLRERLFKSR
>lcl|NZ_CP026520.1_prot_WP_042235640.1_380 [locus_tag=PC41400_RS01960] [protein=MazG-like family protein] [protein_id=WP_042235640.1] [location=complement(438898..439209)] [gbkey=CDS]
MSNREGEVDLAKRVKVIEWLKTEVVDQVAHLLKGIWEGNQAKIIDGLASLISSVYILGRRLGVSFRSLDEAIMDKMKKHR
EEGHQLEDWYGDISALEDHLRKR
>lcl|NZ_CP026520.1_prot_WP_042235643.1_381 [locus_tag=PC41400_RS01965] [protein=cupin domain-containing protein] [protein_id=WP_042235643.1] [location=complement(439322..439675)] [gbkey=CDS]
MKISKQNAEHYVWGDGCDGWYLHKGDETIIHERMPPGTSEERHLHERAAQFFFILTGQVVMETDGEWLELGPQEGLRIAP
GTPHCIHNRSHDAAEFLVISKPGTKGDRIPREELKPD
>lcl|NZ_CP026520.1_prot_WP_082211313.1_382 [gene=rpsR] [locus_tag=PC41400_RS01970] [protein=30S ribosomal protein S18] [protein_id=WP_082211313.1] [location=complement(440017..440325)] [gbkey=CDS]
MSEQQTSSRPQRSEGGDRPERSERKFTPRRGKGGKRRKVCYFTANKITHIDYKDIETLKKFISERGKILPRRVTGTSAKY
QRMLTTAIKRSRTIALLPYTTE
>lcl|NZ_CP026520.1_prot_WP_009673468.1_383 [gene=ssb] [locus_tag=PC41400_RS01975] [protein=single-stranded DNA-binding protein] [protein_id=WP_009673468.1] [location=complement(440345..440827)] [gbkey=CDS]
MLNRVILIGRLTRDPELRYTPAGVAVTQFTLAVDRPFVSSQSKEREADFIPVVTWRQLAETCANYLRKGRLAAVEGRMQV
RSYDNNEGRKVYVTEVIADNVRFLESANSANREEGSYSGGSGGGYGGGGSGSGRNTRDQQQDPFLDDGKPIDISDDDLPF
>lcl|NZ_CP026520.1_prot_WP_042235648.1_384 [gene=rpsF] [locus_tag=PC41400_RS01980] [protein=30S ribosomal protein S6] [protein_id=WP_042235648.1] [location=complement(440866..441153)] [gbkey=CDS]
MRKYEVMYIIRTDIEQEQVQSTVEKFQGIITNGGGEITKHDLMGKRRLAYEIEKFRDGHYVLVHFNAEPAVVAELDRVMK
ISDEIIRYLIVKDVA
>lcl|NZ_CP026520.1_prot_WP_009673434.1_385 [locus_tag=PC41400_RS01985] [protein=YjzC family protein] [protein_id=WP_009673434.1] [location=441397..441579] [gbkey=CDS]
MGEKTQFVPGDKAPNEGRYIEIGENAFHMGINNPEIVHLKKGDSFPDTKNHNRKWKRLYH
>lcl|NZ_CP026520.1_prot_WP_042235651.1_386 [locus_tag=PC41400_RS01990] [protein=hypothetical protein] [protein_id=WP_042235651.1] [location=441597..441878] [gbkey=CDS]
MKKSFTLYGTMLGLLLCLIHYFGHNQDSLYLLFYSLSVPAWFAPMFTNVYEISMGKMFVIYVLTVASWALIGYVIDRFSV
VRKERQGRTGGAA
>lcl|NZ_CP026520.1_prot_WP_042235654.1_387 [locus_tag=PC41400_RS02000] [protein=DUF951 domain-containing protein] [protein_id=WP_042235654.1] [location=complement(442182..442418)] [gbkey=CDS]
MEKKQYELGDIVQMKKQHPCGTNEMEIIRMGMDIRIKCVGCKHSVLVPRAKFESKLKKVLRSSADKGAEVSENGVSGS
>lcl|NZ_CP026520.1_prot_WP_042235657.1_388 [locus_tag=PC41400_RS02005] [protein=mechanosensitive ion channel family protein] [protein_id=WP_042235657.1] [location=complement(442423..443346)] [gbkey=CDS]
MTNMGNRLAAITGLQNPEETVTHFVDDIKKYLLDSQQLIKYGATIVEIVVIYVLSRLIIKVADKLVTRMMETREKSPLKF
DPRRTATIGKLIHNIISYVVNFITILLILAQVGINLGPILAGAGVLGLAIGFGAQSLVKDVITGFFIIFEDQFAVGDVIQ
IDAFRGTVEQIGIRVTRLRSWTGEVHFIPNGNIKQVTNYSINNSLAIVDISIAYESDIDKTIGVLQNTVDRMAEGNENIV
SEPKVLGVQSMGQSEIVLRITAECKPNMQVDVQRKMFAEVKKQLDAHGIEIPCPKTVTYFKGERGAM
>lcl|NZ_CP026520.1_prot_WP_042235660.1_389 [locus_tag=PC41400_RS02010] [protein=DUF3343 domain-containing protein] [protein_id=WP_042235660.1] [location=complement(443360..443611)] [gbkey=CDS]
MGEWLLIAFDSTQQALRAEMLLEYADIEIDLCPTPKGVTAGCALSIQFPAEDYAEVQRIIETEQVEIRGIYFKKESEYVR
MEQ
>lcl|NZ_CP026520.1_prot_WP_042235663.1_390 [gene=yyaC] [locus_tag=PC41400_RS02015] [protein=spore protease YyaC] [protein_id=WP_042235663.1] [location=443766..444380] [gbkey=CDS]
MKPFGFSAKPAAEQPASIKLSHSESGTPAALSAYFSEILKKLPSYQPIVIVCVGTDRSTGDSLGPLVGSNLKRRSGASYS
LYGTLDEPVHAMNLSDTIASIETAHRYPFIVAIDACLGQVSSVGSIQVGHGPLKPGAGVNKELPSVGDIHVTGIVNVGGF
MEYFVLQNTRLNLVMNMADVIAGSLHHALYSVRRSSPTSALTSD
>lcl|NZ_CP026520.1_prot_WP_042235666.1_391 [locus_tag=PC41400_RS02020] [protein=DUF4446 family protein] [protein_id=WP_042235666.1] [location=complement(444355..444849)] [gbkey=CDS]
MGENNIGLPVEPILSVLCLLVLLLLIGMLALWIKLSKLRKQYKKVMNGAGEVNIEQLLVHMQQQIEDQIILQGDTTDRLN
RISQSMKRMKSKVGIHRYNAFSDTGSLMSFTVAILDEYEDGILITGIHSREDTYIYAKPVEKGGSPYSLSPEEKEAISLT
LAQK
>lcl|NZ_CP026520.1_prot_WP_042235668.1_392 [locus_tag=PC41400_RS02025] [protein=aminotransferase class V-fold PLP-dependent enzyme] [protein_id=WP_042235668.1] [location=complement(444916..446079)] [gbkey=CDS]
MTKPLLYFDNAASSWPKPPQVGEAVLRCINECAANPGRGSHAMAVQASRVVFEARKHAAKLFGFKNPNDLAFGSNTTEAL
NLAIKGHVKEGEHVICTAVEHNSVRRPLEYLRRTKNVQVTYLETDEFGRINIDALKENIQSNTSLVVINHSSNLLGSILP
VADIGQLCRDKGICLLVDAAQTAGILPVHVEQMGIDMLAFPGHKGLLGPQGTGGLYIHPRLDLEPLMHGGTGSQSEAIDQ
PSVRPDRYEAGTPNTPGLAGLSEGLKFVLRTGAENIHAKEWGQTQRIMEALRDVPHLRLLGPAIGEERTGIVSFVLEHMD
ASEVAFILDQHYGIAVRAGYHCTPLAHETAGTGKTGAVRASVSYFTTDEEVDTLIEAVKEIADSVIL
>lcl|NZ_CP026520.1_prot_WP_042235671.1_393 [locus_tag=PC41400_RS02030] [protein=ParB/RepB/Spo0J family partition protein] [protein_id=WP_042235671.1] [location=complement(446181..447041)] [gbkey=CDS]
MSKRLGKGLGALIPELEIGEEDKVIEIPLNQLRANPYQPRKHFNEDGIQELASSIKEHGVIQPIIVRTVLKGYEIIAGER
RFRASQASGLATIPAVVKKLSDQQVMEIALIENVQREDLNAMEIAIAYQAIIDQFSLTQEELSVKVGKSRSHIANFLRLL
HLPESIKQYVSRGTLSMGHARAIVGVKDDKVKKELTDACIGGQWSVRQLEEEIKRREELGSEDKEKPKVKDKKKDPYLHQ
AEEQLQDHYRTTVRIKHQNYKGRIELMYYSKDDLERLLEMLQGQIS
>lcl|NZ_CP026520.1_prot_WP_042235674.1_394 [locus_tag=PC41400_RS02035] [protein=AAA family ATPase] [protein_id=WP_042235674.1] [location=complement(447022..447795)] [gbkey=CDS]
MSKTIAITNQKGGVGKTTTSVNLGACLASLGKRVLLVDIDPQGNTTSGIGVNKADVEYCIYDVLINDIHPKDATVATKIP
NLNIIPATIQLAGAEIELVPTISREVRLKKSLQLVEHMYDYILIDCPPSLGILTINSLTASDSVIIPIQCEYYALEGLSQ
LLNTVRLVQKHLNTSLQIEGVLLTMFDARTNLGIQVIEEVKKYFQQKVYQTIIPRNVRLSEAPSHGQSIITYDPRSKGAE
VYLELAKEVISLEQTTR
>lcl|NZ_CP026520.1_prot_WP_042235677.1_395 [gene=noc] [locus_tag=PC41400_RS02040] [protein=nucleoid occlusion protein] [protein_id=WP_042235677.1] [location=complement(448230..449057)] [gbkey=CDS]
MKEQLSKLFGFTERSNMDEVKNIPVQNINPSPYQPRTIFDDDRIDELCQTIRTHGVIQPIVVRVRNNTFELIAGERRLRA
VKKLGLETIPAIVREFNDSQAASIALIENLQREGLTAIEEAAAYQQLIEMHNLTQESLAQRLGKSQSTIANKIRLLNLSE
PVKMALIERKITERHARALLSLDQEELQVKILDEVIAKELNVKQTETRIQFLKEATQIKKSKRVSFSKDVRLALNTIRQS
VEMVASSGLNINTAEQDHEDHYEIIIKIPKQHSAK
>lcl|NZ_CP026520.1_prot_WP_042235680.1_396 [gene=rsmG] [locus_tag=PC41400_RS02045] [protein=16S rRNA (guanine(527)-N(7))-methyltransferase RsmG] [protein_id=WP_042235680.1] [location=complement(449317..450045)] [gbkey=CDS]
MTDLIQEQFVSHLQPHQLSFSDEQLQQFETYYRLLVEWNEKINLTAITERDQVYLKHFYDSLSLAFYVDMEKVETMADIG
SGAGFPSIPLKIAFPHLKVTIVDSLNKRIVFLNELVKELGLKDVTCVHARAEDAARLQQHRDQYDLVTARAVARLSVLNE
FCLPFVKVGGLFVAMKAADAEEELKEAAFSIGLLKGQLKQVDRLELPVEKSVRHFIQISKIAPTPGKYPRKAGLPLKQPL
VK
>lcl|NZ_CP026520.1_prot_WP_042235682.1_397 [gene=mnmG] [locus_tag=PC41400_RS02050] [protein=tRNA uridine-5-carboxymethylaminomethyl(34) synthesis enzyme MnmG] [protein_id=WP_042235682.1] [location=complement(450054..451943)] [gbkey=CDS]
MGYTAGNYEVIVIGAGHAGVESALAAARMGCETLLLTINLDMVAFMPCNPSIGGPAKGHVVREIDALGGEMGRNIDKTYI
QMRMLNTGKGPAVHALRAQADKFLYQHMMKETIESTPNLTLRQGMVEELLVEDGVCKGVVTKTGAEYYGKAVVLTTGTYL
RGKVIMGELMYESGPNNQQPSVRLSQSLRDLGLELVRFKTGTPPRVHKDSIDFSKTEIQPGDENPRFFSYETKEGTQDQL
PCWLTYTTENTHEIINDNLHRAPMFSGAIEGTGPRYCPSIEDKIVRFSDKPKHQIFLEPEGRNTSEYYVQGLSTSMPEDV
QLKILRSITGLEKVEMMRTGYAIEYDAVVPTQLKPTLETKQVEGLFTAGQINGTSGYEEAAGQGIMAGINAACKVQGKDP
VILDRSEGYIGVLIDDLVTKGTPEPYRLLTSRAEYRLLLRHDNADLRLTPIGHKIGLIPEERYQRFMAKKEQIEQELHRL
KTTKVRPEPHVQAILEEAGTVLLNNAVDLLTLLKRPEINYSHIEGMIPLPAELSDEVKEQAEIQVKYAGYIEKQLAQVER
LKKMEKKKIPDDIIYEDIHGIATEAKQKLAKIRPVSIGQASRISGVTPADISILLVYLEHYNQVIAARG
>lcl|NZ_CP026520.1_prot_WP_042235685.1_398 [gene=mnmE] [locus_tag=PC41400_RS02055] [protein=tRNA uridine-5-carboxymethylaminomethyl(34) synthesis GTPase MnmE] [protein_id=WP_042235685.1] [location=complement(451966..453342)] [gbkey=CDS]
MISDTIAAIATPLGEGGIAVVRVSGDEAVKLVEPLFRSKVKLSEAQTHTIHYGHIIHPSSGDKIEEVLISVMRAPRSFTT
EDVVEINCHGGIVSVKRVMDLLLEQGIRVAEPGEFTKRAFLNGRIDLTQAEAVIDLIRSKSDRAFQIALKQVDGNLSKAI
GELRYQLVELMAHIEVNIDYPEHDVEEMTNAFIRERCIQVSSRIDELLRTAEHGKILREGIVTAIIGKPNVGKSSLLNTL
AQENRAIVTDIPGTTRDVIEEYVNVGGIPLKLLDTAGIRETSDLVEQIGVERSRTALSEADLILLVLNSSEPLLPEELAM
IEELRQRQTIVILNKTDLPLQIDKEEIEAKFPQGRVVSLSLLEGSGLAELEKAISAIFFEGNLESTDLTYVSNVRHIGLL
KQAKGSLREAMEANDAYVPIDMIQIDIRAAWEHLGEIIGDSVAESLIDQIFSQFCLGK
>lcl|NZ_CP026520.1_prot_WP_042235688.1_399 [gene=jag] [locus_tag=PC41400_RS02060] [protein=RNA-binding cell elongation regulator Jag/EloR] [protein_id=WP_042235688.1] [location=complement(453431..454048)] [gbkey=CDS]
MKKVIATGKTIDDAIKAGLAQLSTTQDRVHVTVLEQPSKGFLGLIGTKEAKVELEKIPDPIEEAVAFLKDLFDTMGLAIS
IDQKQTKEGHLFNLSGSELGILIGKRGQTLDALQYLVNIVANRFSNAHVRILLDAEQFRDRRKQTLQELAHRLAGRVVRS
RKELILEPMNAQERKIIHAELQSHPSVKTYSKGDEPNRRVVITLR
>lcl|NZ_CP026520.1_prot_WP_042235691.1_400 [locus_tag=PC41400_RS02065] [protein=YidC/Oxa1 family membrane protein insertase] [protein_id=WP_042235691.1] [location=complement(454045..454800)] [gbkey=CDS]
MIRRTYKYAPLLLLLFVLTGCGTSPTSYPPLDPATAGFFTKYFIIPLSDLLDFFANSIGNSYGISILIVTIIIRVIVLPL
TLKQYKSSKRMQEVQPEMAKIREKFKDNPQKQQEETMKLFQKHGVNPLAGCFPILIQMPILLALYQAIVRNPHIFSHQFL
WMELGKPDPFYVLPVLAAATTFIQQKVMSAQNPMNKQMQSIMFIFPVMIFVMSMSFASALPLYWIYSNVFTIVQTYFLYG
RSPKTKGGQAA
>lcl|NZ_CP026520.1_prot_WP_009673476.1_401 [gene=rnpA] [locus_tag=PC41400_RS02070] [protein=ribonuclease P protein component] [protein_id=WP_009673476.1] [location=complement(454905..455252)] [gbkey=CDS]
MEKKLRLAKREDFQKVYRYGKSAANHQFVLYYMPRRENEHFRLGISASKKIGNAVIRNRLRRVLKEIIRLNEERFPGGYD
YIVIVRKPAVEMDFHEMEKSIYHVFKRAGLLRKNS
>lcl|NZ_CP026520.1_prot_WP_009673467.1_402 [gene=rpmH] [locus_tag=PC41400_RS02075] [protein=50S ribosomal protein L34] [protein_id=WP_009673467.1] [location=complement(455391..455525)] [gbkey=CDS]
MRPTFNPNTRTRKKNHGFRKRMSTKNGRKVLQARRQKGRKVLSA
>lcl|NZ_CP026520.1_prot_WP_042235694.1_403 [gene=dnaA] [locus_tag=PC41400_RS02080] [protein=chromosomal replication initiator protein DnaA] [protein_id=WP_042235694.1] [location=456003..457367] [gbkey=CDS]
MDSHTTDSWQQVLSIIQKKLSKPSFDTWLKSTRATVFTDTTVVISAPNNFAKEWLESRYAKMIGTTIYEFLGKQVEVKFV
NESDEKSPVAVQTTPPEIPKEPAAPVQEETFHHMMNSRYTFDTFVIGAGNRFAHAASLAVAEAPAKAYNPLFLYGGVGLG
KTHLMHAIGHYVLEHNPRARVLYISSEKFTNEFINAIRDNRGEGFRNKYRSIDVLLIDDIQFLAGKEQTQEEFFHTFNAL
HEEGKQIIISSDRPPKEIPTLEDRLRSRFEWGLITDIQPPDLETRIAILRKKAKAENLEIPNEAMVYIANQIDTNIRELE
GALIRVVAYSSLSNQDITTHLAAEALKDIIPSSRPRVITMQDIQQKVGEFYGLKMEDFKARKRTKAVAFPRQVAMYLSRE
MTDFSLPKIGEAFGGRDHTTVIHAHEKISEALKTDQELFKIIHSLTEKIKNPSS
>lcl|NZ_CP026520.1_prot_WP_042235697.1_404 [gene=dnaN] [locus_tag=PC41400_RS02085] [protein=DNA polymerase III subunit beta] [protein_id=WP_042235697.1] [location=457536..458678] [gbkey=CDS]
MKLTILKDQLNESIQHVSKAISSKTTIPILTGIKIDATSEGVTLTASDTDISIQSFMPTETAELKVIELFQPGSVVLPAK
FFMEMIRKLPSQTIEIEVKDHFQTTIRSGSSEVQIMGLDPEEYPLLPQIEESRTIRIPSDLLKTMIKQTSFSVSTNETTP
ILTGVLWQISDNKLKFIACDRHRLASREINLDLEPGQKFHPIVISGRTLNELSKILPDQNSLIDIVIADNQVLFKMNSIL
FYTRILDGTYPDTSKLIPQSFQTELVVGTKELADAIDRAYLLSREDKTNIVKLVMKEDETIEVSSSSTELGKVTEQLQLQ
KINGELLRISFNSKYMLDALKVMDSDHIHIGFTGAMQPIIIRPEEQQNILQLILPYRTTN
>lcl|NZ_CP026520.1_prot_WP_042235700.1_405 [gene=yaaA] [locus_tag=PC41400_RS02090] [protein=S4 domain-containing protein YaaA] [protein_id=WP_042235700.1] [location=458718..458933] [gbkey=CDS]
MKEVTIRTEYITLGQFLKLADCIQTGGQAKSFLQEAHIEVNGENENRRGRKLYSGDKVAVEGCGEFVVSRE
>lcl|NZ_CP026520.1_prot_WP_042235702.1_406 [gene=recF] [locus_tag=PC41400_RS02095] [protein=DNA replication/repair protein RecF] [protein_id=WP_042235702.1] [location=458961..460079] [gbkey=CDS]
MFVKRLLLEHYRNYPHVEISTDRNVNIFVGPNAQGKTNLLESIYVLALTKSHRTHQEKELIQWQEDHARLFGEVERKYGM
STLELSISGKGKKAKVNGLEQKKLSQYVGALNVVMFAPEDLEIVKGSPGIRRRFLDMEIGQVYPSYLYDLTQYQKVLVQR
NNLLKQSYPPSAANSSMLDVWNEQLAAFGVKIMKKRQSFIKKLQSWAEQIHQGITNGKEMLRVEYAPSFADGHFEDESVL
LDRFMIKLSQIKEQEMRRGVSLIGPHRDDLLFCINDKDVQTYGSQGQQRTTALSLKLAEIELIREEVGEYPVLLLDDVLS
ELDEYRQTQLIQTFQKKVQTFITTTGLESVHLDQLDHAAVFHVSSGSVIDRS
>lcl|NZ_CP026520.1_prot_WP_042235705.1_407 [locus_tag=PC41400_RS02100] [protein=DUF370 domain-containing protein] [protein_id=WP_042235705.1] [location=460086..460334] [gbkey=CDS]
MFIHLGGEKIIRSSELIAIFDLSIEKSSKISKQFIAEATRSKKIEIIGEEECKSLVVTQGKVYYSPISSMTLKKRANQLL
TN
>lcl|NZ_CP026520.1_prot_WP_042235708.1_408 [gene=gyrB] [locus_tag=PC41400_RS02105] [protein=DNA topoisomerase (ATP-hydrolyzing) subunit B] [protein_id=WP_042235708.1] [location=460369..462285] [gbkey=CDS]
MTLNQNSYDENQIQVLEGLEAVRKRPGMYIGSTSSRGLHHLVWEVVDNSIDEALAGYCTRIDVIVHKDNRITVIDNGRGI
PVGEHAKMKRPALEVVLTVLHAGGKFGGDDSGYKVSGGLHGVGISVVNALSSHLHVRVKLHGKIHEQEYRQGKPQYDLRI
IGETEETGTTVTFTPDPEIFTETTEYDYEILQSRIRELAFLNKGIEINLTDERTDTSESFMYEGGIKSFVEHLNRNKEAL
HQPPIYVEGSKENIQVEVAIQYNDSYTENIYSFANNINTHEGGTHESGFKSALTRILNDYARKTNAIKESGSNLSGDDVR
EGLAAIISVKIPEPQFEGQTKTKLGNSEVRGIVDSLFAEKLLEFLDENPAIAKKIVEKGLQASRAREAARKARELTRRKS
VLEVSSLPGKLADCSSKDASISELYIVEGDSAGGSAKQGRDRHFQAILPIRGKILNVEKARLDKILSNLEIRAIITAMGT
GISDDFDITKARYHKLIIMTDADVDGAHIRTLLLTYFYRYMRKLIEVGYVYIAQPPLYKLERGKVARYAYSDKQRDEIMK
EFGEGAKVNVQRYKGLGEMDAEQLWDTTMDPETRTLLQVTIQDAMDADSIFDTLMGDNVEPRRNFIHQHAKYVKNLDI
>lcl|NZ_CP026520.1_prot_WP_042235711.1_409 [gene=gyrA] [locus_tag=PC41400_RS02110] [protein=DNA gyrase subunit A] [protein_id=WP_042235711.1] [location=462350..464875] [gbkey=CDS]
MTEEMRSQVVEVDISSEMRTSFLDYAMSIIVSRALPDVRDGLKPVHRRILYAMSELGMYSDKPHKKSARIVGEVIGKYHP
HGDSPVYETMVRMAQDFSLRYMLIDGHGNFGSVDGDMAAAMRYTEARLSKIAMEMLRDINKETIDYIPNYDGEEQEPVVL
PARFPNLLVNGSSGIAVGMATNIPPHNLGEVIDGIQQMIENPDVTPLELMQSIKGPDFPTGAYILGREGIRQAYATGRGS
VTMRAKADIEEVNNKARIIVHEIPYQVNKARLIEKIAELVRDKKIDGITDLRDESDRNGMRIVMELRRDVNPNVVLNNLY
KHTALQSNFGIIMLALVNGEPRVLNLREMIYYYLEHQKVIIRRRTEYDLRKAEARAHILEGLRIALDNLDEVIALIRSSR
TTEAAREGLMTTFNLSHDQAQAILDMRLQRLTGLEREKIEQEYAELLQKIAELKAILADEQLILAIISEELNEIKQSYGD
ERRSEITKGEDSIEDEDLIPQEDVVITITHTGYIKRLPVNTYRSQKRGGRGIVGMDTKTDDFVNHLFVTNTHHYLLFFTD
RGKVYKLKAYEIPDLSRTARGTPIINLIQIEQGETVSAVIPVQEFSADNYLFFATKQGIVKKTPLDDYINIRKGGLIAIN
LRDDDSLVGVKLTDGKRDIIMGTRDGMSIRFPENDVRSMGRSATGVKGIQIGPEDTVIDMDVVDEDASVLIVTSKGFGKR
TPIAEYRVQSRGGKGIKTLNVTTKNGAVVGLKVVNNEEDLMIITASGTLIRTSISGISEMGRNTQGVRLINIREDDEVAT
VTHVAKSEESQDDEFNEEDQEQMASEQTDAQVDNPEEDTEI
>lcl|NZ_CP026520.1_prot_WP_042235713.1_410 [locus_tag=PC41400_RS02115] [protein=HD-GYP domain-containing protein] [protein_id=WP_042235713.1] [location=464965..466038] [gbkey=CDS]
MRTVSVSQIKAGDKLLEAVNSRFGNALFYKGRILTERDAEILKAFLISTVTIDNGTSNSGQESASAVASADIDTKQISSF
DLSYDKMLQLLKKAFGLVLGNSALPVLEMRNELHKLIEHIEGYQVLTFTPRQFQLQDFVHHNSIMVSLTSYLLAKWHGLP
QKDWMQIALAGLLHDIGNAKIDQALLVKPTKLTPIELEEMRKHTLYGYNVLKPIAALNEGVKLCSLQHHEREDTSGYPLG
LKSDQIHIYSKVVAVADIFHAMTSVRYHQEATSPYLVLEQLSKDSFGKVDPAIVNTFINKVTSFHIGTVVKLNDNRVGEI
VFSDRSYPTRPWVNINGTIVNLTLERNLYIQEVIEAR
>lcl|NZ_CP026520.1_prot_WP_042233979.1_411 [gene=guaB] [locus_tag=PC41400_RS02135] [protein=IMP dehydrogenase] [protein_id=WP_042233979.1] [location=471542..472999] [gbkey=CDS]
MWDAKFAKEGLTFDDVLLVPRKSNVFGKEIDVSTKLAPKVKLNIPLLSSAMDTVTEAALAIAIAREGGIGIIHKNMSIEL
QAGEVDRVKRSESGVITNPFHLTPQHHVYDAEALMAKFRISGVPIVDENKKLVGILTNRDLRFVHDYSIKIEEVMTKENL
VTAPVGTTLEQAEGILQQHKIEKLPLVDDSFVLKGLITIKDIEKAIQFPNAGKDEFGRLLCGAAVGVSKDVMERSAALVE
AGVDVLVVDSAHGHHINILETVRKLRNQYPELPLIAGNVATGDGTRDLIEAGASIVKVGIGPGSICTTRVIAGIGVPQIT
AIYDCASVAREYNVPIIADGGIKYSGDVVKAIAAGASAVMLGSLFAGTEESPGESEIYQGRRFKVYRGMGSLGAMKEGSK
DRYFQENETKLVPEGIEGRVAYKGPLADTIYQIVGGLRSGMGYCGAPDIETLINDTTFVRISGAGLRESHPHDVQITKEA
PNYSL
>lcl|NZ_CP026520.1_prot_WP_241688874.1_412 [locus_tag=PC41400_RS02140] [protein=D-alanyl-D-alanine carboxypeptidase family protein] [protein_id=WP_241688874.1] [location=473109..474410] [gbkey=CDS]
MCELKKFGFWKRKVSLALVFAMLQVSFLHIFAVTALAADGEPELDVKSAILMEASTGQILYEKNADEVLPPASMSKMMTE
YLVMEAIHNGKLKWEDTATASKYAASAIGSGQLIAEGETLTIKDLFYAMSIYSSNDASIVLAEKLGGSEEGFAKLMNDKA
RELGMSEKAQFINATGLSRADIKENAPKSIQGETVMSAKDVAILARHLVTEHKEVLEYTKVPSKKLRESDKSPMINYNWM
LEGNSSNVNFKKYAYQGLDGLKTGHTDEARYCFTGTAERNGMRLISVVMGAKTEPERFNQTRKVLDYGFTNFEFKTLAKA
GTALDAQKTVPITKGVELEAPAVLKNDVAFAVKKGSADAKLETKVDLLAADKLVAPLKKGDKLGTATLSFGKNSETVDLV
AENDMEKAGWFRMMMRAIKNLFVDLFNGVKGLF
>lcl|NZ_CP026520.1_prot_WP_009677085.1_413 [gene=pdxS] [locus_tag=PC41400_RS02145] [protein=pyridoxal 5'-phosphate synthase lyase subunit PdxS] [protein_id=WP_009677085.1] [location=474526..475407] [gbkey=CDS]
METGTSRVKRGMAEMQKGGVIMDVMNAEQAKVAEAAGASAVMALERVPSDIRAAGGVARMADPTVLEEVMKVVSIPVMAK
ARIGHIVEARVLEALGADYIDESEVLTPADEVFHIDKKDFTVPFVCGAKDIGEALRRINEGASMLRTKGEPGTGNIVEAV
RHMRLIQSQIRKIQSLAKDELYAEAKNLGAPYDLVLYIHENGKLPVVNFAAGGVATPSDAALMMELGADGVFVGSGIFKS
DSPEKFARAIVEATTHYKDYKLIAEVSKNLGAPMKGIEISKLHASERMQDRGW
>lcl|NZ_CP026520.1_prot_WP_042233975.1_414 [gene=pdxT] [locus_tag=PC41400_RS02150] [protein=pyridoxal 5'-phosphate synthase glutaminase subunit PdxT] [protein_id=WP_042233975.1] [location=475437..476009] [gbkey=CDS]
MKVGVLALQGAVAEHIRLIGAVGGEGVVVKRAEQLAELDGLIIPGGESTTIGKLMRRYGFIEAIRDFSNQGKAVFGTCAG
LIVIADKIAGQEEAHLGLMNMTVQRNAFGRQRESFETDLPIKGIDRPVRAVFIRAPLIDQVGDGVDVLSEYNGQIVAARQ
GHLLAASFHPELTDDSSMHAYFLDMIRETR
>lcl|NZ_CP026520.1_prot_WP_042233972.1_415 [gene=serS] [locus_tag=PC41400_RS02155] [protein=serine--tRNA ligase] [protein_id=WP_042233972.1] [location=476099..477382] [gbkey=CDS]
MFDIKLIRTDIETVQTAMKNRGGKQLEEVTRIAELDVKRRELLQEVEQLKNRRNTVSQEVAMRKKAKEDAEDLIAEMKGV
GDRIKELDEELRTLEVELNHLLLSIPNVPHESVPVGASEEDNIEIRRNGNVPAFGFEAKAHWDLAMDLGILDFEAAGKVT
GSRFVFYKGLGARLERALMNFMMDLHSTEHGYEEMLPPYIVNRDSLTGTGQLPKFEEDVFHIENSEYFLIPTAEVPVTNY
HRDDILAAEQLPRKFVAFSACFRSEAGAAGRDTRGLIRQHQFNKVELVKLVKPEESYQELEELTQNAERVLQLLKLPYRV
LTLCTGDIGFTSAKTYDLEVWLPSSEAYREISSCSNFEDFQARRANIRFRRDAKAKPEFVHTLNGSGLAIGRTVAAILEN
YQQEDGSVTVPEVLVPYMNGVTVLRKN
>lcl|NZ_CP026520.1_prot_WP_042233969.1_416 [locus_tag=PC41400_RS02165] [protein=hypothetical protein] [protein_id=WP_042233969.1] [location=477766..478116] [gbkey=CDS]
MIVKTEKPSLERDQLAKAWMEELPNTLPESDHARVWPDEADPQAIRIHITNAGRTHYTFDFKCTYVDSREVKVELVDVER
DDVHADETSEIIQGLVKDYVRHIHECAQVLQELTHA
>lcl|NZ_CP026520.1_prot_WP_016423741.1_417 [locus_tag=PC41400_RS29635] [protein=hypothetical protein] [protein_id=WP_016423741.1] [location=478149..478295] [gbkey=CDS]
MTRQRTRDKNLQNVAKDLNKSAVNSHDAQQIQQQTNDRRHQDALNHEE
>lcl|NZ_CP026520.1_prot_WP_084706672.1_418 [locus_tag=PC41400_RS02170] [protein=small acid-soluble spore protein P] [protein_id=WP_084706672.1] [location=478337..478486] [gbkey=CDS]
MSKPKALPVGDGVTQKELERYNTPANEPLSGSKKVKKRNHSRQNHGEGR
>lcl|NZ_CP026520.1_prot_WP_042233967.1_419 [locus_tag=PC41400_RS02175] [protein=thioredoxin family protein] [protein_id=WP_042233967.1] [location=478870..479451] [gbkey=CDS]
MGFTLGIGDGARDFRLQGTDGNIYTLDSFRHAKALVVFFTCNHCPYVLGSEEITRRTAEQFTEQGVAFVAINSNSSQTNP
EDSFENMVERMEEKKFPWVYLHDKGQEVALAYGALRTPHFFVFNDRRKLVYTGRGVDNPRETANMTVNNLQEALEAISAG
KPVPTPVTNPIGCNVKWEGRDKHWMPEEACDLV
>lcl|NZ_CP026520.1_prot_WP_042233966.1_420 [locus_tag=PC41400_RS02180] [protein=hypothetical protein] [protein_id=WP_042233966.1] [location=complement(479548..480678)] [gbkey=CDS]
MNPRSNTVALLLSFIPGLGHLYLKRIFRAFLYGAGFFGPVAMAFLIVVEGHGPKDFAVLLLLFAAACWVVNMVDMFITLG
SRPSYPPPGYHEYGNPGHDFGHGPGPGAVPEFNPAVYEWQRREERRKIILASFIPGLGHYMLGMMYRGLSAMTFFIGGAI
VIFFGASLTRTGEFLFFLLALPVLWIYILFDAIRQLEKKQAGMELTDRTFFDDFSESRQEGRRSKTIATVLAVFPGAAHL
YLGLQMRGLQIMAGFLFSIYIMDVLRLSLFLFLIPIFWFYSFFDALQLISRQEFGYVEDKPIFKGIKRHQRKIGFLIILV
GLYYLTNEVLLDVLRRYYPDWHWPYSISSYFQTFIVALVLIGGGLTLLFNSPNRKS
>lcl|NZ_CP026520.1_prot_WP_042233964.1_421 [locus_tag=PC41400_RS02185] [protein=hypothetical protein] [protein_id=WP_042233964.1] [location=complement(480668..481216)] [gbkey=CDS]
MSHRSRDEWKLYIAGEITDDRQRGLWEDHLYECDLCLELYTECLEFLSPTLPLPDETETERMIDQILARSGSLPVVREAA
PMAARKQEESEGNLDRTEAKKHRKARSPRRTALTHYAIAAAATFLLMTTGVFGHMTDQFSLVRTEQKKPLATPVSDKLME
QTFAFFDSLHAQKPKGGHPLEP
>lcl|NZ_CP026520.1_prot_WP_009676999.1_422 [locus_tag=PC41400_RS02190] [protein=sigma-70 family RNA polymerase sigma factor] [protein_id=WP_009676999.1] [location=complement(481213..481737)] [gbkey=CDS]
MLAGDEEAFREVVRTYRNYVFQIVFSVVRHPKDAEDVMQEVFLKMYTSLPQTEVKGLKAWISRVAVNKAIDYKRKAYRQR
ETATDGMEEAMPPPSLSGSDTEDPLLSKELQALVMRKLEELPENYRDVIIAFYYEDKSYQQIADEQEVTVQTIKSKLYRA
KQWIKQHWKEEEFE
>lcl|NZ_CP026520.1_prot_WP_042233962.1_423 [locus_tag=PC41400_RS02195] [protein=ABC transporter ATP-binding protein] [protein_id=WP_042233962.1] [location=complement(481907..482749)] [gbkey=CDS]
MIRLNQLVRLYEGHIRLHVDQLTLAPGLILLVGKNGAGKSTLLRLLATADFPDGGEISYLEGTVKDQLPQIRSRIGYVPT
DVELHEDMTPIRLMNYMCLLKGADPPQGEQLLRELHLEDVKRKRIGSLSQGMKQRLAIGQALLGSPRYLLLDEPLNYLDS
IERKTVIRLLYRHARGCIVLIAAHELNEWQEADQLLWLNAGRPEFYGTPGAWRQVPLGVWQGILPHADACPFDEACVMIR
KLVGDHIEWRVFSRECPAPGFVQVPLTTEDAYFIRSRRLL
>lcl|NZ_CP026520.1_prot_WP_042233959.1_424 [locus_tag=PC41400_RS02200] [protein=hypothetical protein] [protein_id=WP_042233959.1] [location=complement(482746..483582)] [gbkey=CDS]
MYEPDDKELDQLEKDTAVLLNGYLVKTPDRIDTANLLGRLQSHFDAIRENSPGRARPEHSADLRPSLWKLCCHQLFMYNK
WFWLLSLLLFAMLLLLNSPYTIAGTLPGRSSYSLAFPLLLVCGFLYQYRSWNEGVRMIESVTPYPPALLMFSRLIVLLVT
NAVLGLIGSLYTFATMSYSRSWVTLPFLFDWIAPLLLVTGLTAYFMMKRGMLSAIMAALTGWGVELFMEEWLRRELVGQV
WNWALILQAVLIAAGVLFLLSAYRNSMRLTSMKGWRSA
>lcl|NZ_CP026520.1_prot_WP_042233958.1_425 [locus_tag=PC41400_RS02205] [protein=RNA polymerase sigma factor] [protein_id=WP_042233958.1] [location=complement(483575..484192)] [gbkey=CDS]
MLTDEELVQAMARGDQAAFEAFVHRYHAPLQHYLERTLHDPRKAEDLVQEVFVRLLKQLQQQKIPDHIRPWMYRVALNLC
RDYWKSLAYRNEQQLTAEPPEAQDIQPSVVEIYERQEARKEIIGALRSLPEVQKEIVILRFFHDLKLQEITEVLQIPLST
TKTHLYSGLRKLKALLGPKAGSLTDTGGGRKENRGKPTKEGTPYV
>lcl|NZ_CP026520.1_prot_WP_241688776.1_426 [locus_tag=PC41400_RS02210] [protein=hypothetical protein] [protein_id=WP_241688776.1] [location=complement(484236..484952)] [gbkey=CDS]
MSLWLQDLRGFKSIWGLPPLLYVLMLGYAIWSSDPEQPGNALFYGELAFYPLVVMLCAALFHRELGGPMEIYATLPQSLT
GMAARKGLILLAVCTGYQAAWFTAYQAKFGAVTTLVYSYGPAGQSSMCTLHSFMPLLLQMLPALLFIAVLTLTVMLAVKK
LYAGLFAGLALWMLDILSHGSWLPYFTIYTVHIQEQSKEIPNLFLPNRLVLLSATLILTVILNYTVQDRSRWIVSEEE
>lcl|NZ_CP026520.1_prot_WP_042233954.1_427 [locus_tag=PC41400_RS02215] [protein=hypothetical protein] [protein_id=WP_042233954.1] [location=complement(484949..487159)] [gbkey=CDS]
MNRWLRAFSLELRLLRSNPVLLVLPVLYGAWMIYSLARIAPPVSQDLYLYVYDFHKVKHTLTLGAAMLLGILLIRRDTAK
MAYEWTAGLPVSPAVLFSVKYAAGLLYLSLFTAAMAVVYVYFALHYGIARPIMFRELSFFAVQYEWSYMVTLALAMLLAI
AIRNRIVYLIGFCAWVFGTFFIDQFIIERLGVLELRVLKPFHLSGMTLASFYANEAWGPWITNRETWLARWFVLSFTLVL
LTVCVTILKWVRPSGTRARWAGALILFTVLSAGSYYTYVGFWTPHFAAYREQLRNAPTESVYSQRTITQFTVSSYKMELA
PISSDGLQASVEVSLRPADLPQGDVSFTLHDYFHVEQLLLDHQPVSFTRRGNLISVESRYFDSSKSSQTLTFNYKSRGVN
WTYGSQQGEQIISFVMDPYLYLPNSAAWYPLPNNGPLYLTDERKKELFYGGPQPLAQPADFDVTFKDFSNPIYGSITEKQ
GAPRIQHFEQKGVKYLTFIAGNLIQFSDAKGSLKLQTTPGNRWEADLFWKDILRMKAYYEALLQTNLSAVRNIVYMPFGN
LASLDSRQSAVAGSSYYINESKNHNLDDYQRIRTTYALLFGDDSYQSSYAEVSRDASGNTDETVKPSIFMEIRAAFLYLY
YRDGLQLTEEQIRSVQFSGQFLPKDRTTGDQSLGQQIKNHILLQVREALDAGRTEDVKKVLASFYKKGLGAPDAPFRYPV
VTLDDWNREWAKVFGS
>lcl|NZ_CP026520.1_prot_WP_042233952.1_428 [locus_tag=PC41400_RS02220] [protein=ABC transporter ATP-binding protein] [protein_id=WP_042233952.1] [location=complement(487152..488021)] [gbkey=CDS]
MIEIRNLSKTYKHHPALTGVNLEIGQEMFGLLGPNGAGKTTLMRILSTLLEPTEGTVRVGGLNLLSDAEHIRKMTGYLPQ
YFHLYPQLTAYEFLDYVAVMKGIKSARERKSQIGEILQQVNLADKAGAKIKTFSGGMKQRLGIAQALLGDPKVLIVDEPT
AGLDPEERVRFRNLLSRSSLSRTVLLSTHIVADIESSCSRVAVLHRGRIAFAGSLAGLQECAEGLVWEAELSEREFAAYE
PHEVVTARRTDHGMLCRIVSATPPAGNAAAVKPTLEDGYLALIGGDRRE
>lcl|NZ_CP026520.1_prot_WP_042233950.1_429 [locus_tag=PC41400_RS02230] [protein=S-layer homology domain-containing protein] [protein_id=WP_042233950.1] [location=488503..490050] [gbkey=CDS]
MRTRTGKMLIGGFLTAALLASGLGGNGYKAAAASQTSPSYKSTFKDTKGHWAEETIRWAYEHKMTDGYADGTFRPDKAVT
EPEFLALLLRAYNKVGSETAAGSQWFDPYYTKAAIYSWPLLKNTTSPYLRGDAARLIAATQGNLLSTKEAVQRLLDQGLS
QGKTDGSENGFRPDDALTRAEALKFLQNLLGVQSDLIAVAYSEVPNNSGSAGGTAPGGTVAVVDKPDSNGSVKTESLEAR
HFVIEGAAVGDTEADLIKRLGQPDRKDTGGSGLTWYIYNTTDYKHYLQAGIKNGRVAALYTVTGNWTEANGVTTLMNEAA
VRTKMTGAKAVGGTAKNAVSFETDGLNVSFYFDTNDNGKLIGMLVQDTSMSAANSSSGTLASRSEDQRTAYEKQTFDLTN
AIRVRQGKAPFAWDASIASTARKHSEDMAVNGYFDHTNKQGLSPFDRMEKDGLKFWSAAENIASGYSGPIEVVNGWMNSS
GHRKNIMGDTTRLGVGVYFGGSMNVYYTQNFYTPR
>lcl|NZ_CP026520.1_prot_WP_042233946.1_430 [locus_tag=PC41400_RS02235] [protein=FAD-dependent oxidoreductase] [protein_id=WP_042233946.1] [location=490284..492272] [gbkey=CDS]
MTQTTDSRKKENQSPTYPRKGMGLISAFAFLLLISLSWWAWSAWKGTDMAKPGQSQSVKKVASIKAPDSEYDVIVAGTDP
EGIAAAVSAARNGQKTLLVENRARDVLGGLFTEGWLNSLDNSFAPNEEGKSDAGKFLNEGIFLEWRKGVEGTSFDVTTAA
NAFYDLVRKEKKVDLLMHAKSFTPMVNGSQTVTGLNITTADGKEHKVSARAVIDATQDGDVAAAAGVPFTIGRQDLGDDK
SLMAVTLVFRLKGMTEEIWQQMKKNGENDPHLLRGMGADAMSGWGYSAMWDYKSTNPERVRMRGLNIGRQNDGSILINAL
QIFGVNPLDPASVKEGMEIGRAELPHVLDYMKKEFPELASLELDGSAPELYVRETRHMIGEYRLTMVDALDNRDQWDKIA
YGSYNLDIQSTDAKYRGDIVLKPSQYAVPFRALVPQKINGLLVVGRSASFDSLPHGSARTVPVGMASGQAAGAAAAVATK
NKVSFRELSLSKPLVTELQETLVKQGVRLEPYKFAAPPYTKHKAYEGLKAAVSLGIAAGGEKNEFQLDEPSNPRRFANHF
NTMKTVLGRTYPKAVGGSADAAVKGLTKPGDIALTWEQAAWTISQAVGAKVRRSESAAYVLEQKLVEPATFQAIENKSKL
TNGDAYMLLRDVLKAQAGVVYR
>lcl|NZ_CP026520.1_prot_WP_241688777.1_431 [locus_tag=PC41400_RS02240] [protein=EAL domain-containing protein] [protein_id=WP_241688777.1] [location=492589..494625] [gbkey=CDS]
MIKVFVFLTTILSAVFLRKSVVLQHILASIAYITLFTLLHNIDYFKLPSIMYTLLLLYAFLLPGLWVPTIVGAGFLLFLL
TYSSTVQFDYLRANLVGVVFNTVVYAALSYFFRQMIIERDRLKESENRYRSLIEICPEPIVVQQKGKLVYANPAAVQSLG
AEDVNKLLGRSALRFLHPESRKNALEHVNEVLEKGVSARMEQRYVRLDGQVRSCELVLVATNYGGKPACMSIGKDITERK
KAEEKINHMAYHDTLTGLPNRSYLNECLYMAIERAEQTGSQLAVLFLDLDRFKNINDTVGHIYGDLLLRQVAERLRACLG
PSDTVFRHSGDEFVILLEGVTEDGVRVQAQAVLDQFKQPFQLHGHEVYTTTSIGISIYPQDGMDRDVLIKHADMAMYLSK
ENGKNNYQYYRVHNNRLARKLKLENGLRRALEKQLFTLHYQPQINLASGEIVGMEALIRWMDPELGVVTPGEFIPIAEET
GLIVPIGKWVLHTACHQYKQWREAGLPEIRLAVNISVRQFQDKQLVSTIQSVLEEFDMNPGHLELEITESIMQNLDQSLV
ILTALKKIGVKLSIDDFGTGYSSLFLLDKLPIDHIKIDRTFIEEITFNANTEAIVRTIIDMGRHLKFQLIGEGIEHERQI
HFLQEHQCTIGQGFFISPPLSSEKAGELLSRKQPFVQV
>lcl|NZ_CP026520.1_prot_WP_042233940.1_432 [locus_tag=PC41400_RS02245] [protein=AMP-binding protein] [protein_id=WP_042233940.1] [location=complement(494729..495958)] [gbkey=CDS]
MSVKFTALMERLEQSPFYRQHLSAEEREIRDISRIGELPLTTKDHLREAGPWNLLAVPPEAVCQYFETFGTTGLPASAWY
TVRDLEAGGRQIGDSGLRLRPGDRLLNRFPYALSLPAHLVQQAARQAGACVLPASSRNTMTPYPRVIELLRKLKVTVMTG
LPREMEILAETARLHGLGRREDFPHLRAIGVAGELLAEARRKKIEELWGVPVYNLYGSTETANIAAMCEHGVLHVAEEDF
FVEVLQEDLSASVEPGGGKGLAVITTLTHEGSPLLRYLNGDVISLEEVSCPCGRSGHRLTHYGRQLDRLEVAGRILEPAD
IHDAVYSLPDLPDAWKVTPAQGGLRFTFEWPDKDAGVLPRLAAILAEKLQLPVVVEPGEPGALFDRRLLLERQVSTKPLY
IVKPAEVVL
>lcl|NZ_CP026520.1_prot_WP_042233938.1_433 [locus_tag=PC41400_RS02250] [protein=3-dehydroquinate synthase II family protein] [protein_id=WP_042233938.1] [location=complement(496025..497314)] [gbkey=CDS]
MSKALKETPDINPKASNDSTTQPTEKPAETKTNGTAAPVSSLNQAKNKPIAPVRSKYAWFDISSLTAPEQWGAIVQALPH
QKFTGVVANEQQVKQLPPTLDKITYVVGTPSPDKLTALNEAYDIVILEPSVIGSSWFRRNREAITCKIGTSVNVTDPESL
QLAVAMTHEVSLLIVEFRDDTKIPLEIVLADAQNFGTEVVMKVRDDIEAKVVFGVLECGADGVLIASGDLGQVYAVGEVI
TSTGQKGGQPLQKLTIVRTEHVGMGDRACIDLTSYLGLDEGVLLGSFSCGGILACSETHPLPYMPTRPFRINAGSLHTYV
LAPDNRTWYLSDLRSGMEVLVVSTSGKARRVSVGRVKIERRPLLYIEARAEDGTTLNVIMQEDWHVRVFGTEGQPINITT
LKPGDQVLGYTMDPGRHVGVKVDELIIEQ
>lcl|NZ_CP026520.1_prot_WP_042233936.1_434 [locus_tag=PC41400_RS02255] [protein=AMP-binding protein] [protein_id=WP_042233936.1] [location=complement(497333..498682)] [gbkey=CDS]
MAVTARSQIELMSGQKAADRFEPHVRWLNSLTERGMSVENLSQAEVDAYHRRALRSLLQFVYETNEYYKQRIDITGLTPE
EASNPALFAQLPFLTKEDLRNDPRCIICSPDLAHIHTSTGTTGGKPIFIGHTLNDLYRLDTAPRYPSLMQNTRNTVVGNA
LPYEMSSSGLSFHRVFQYMQDCCVLPVGKGGAYSQPDKALECMLAWDMQTLVIAPSYAVTLAEHAQKLGMDLHRDFNLQH
LFLTGEGASPALRERIERIWSTPVTTWYGSLEVGLIGIECLEHQGYHLAEGHNYTEIVHPVTGEPQAPGIVGEVVLTTLM
RDGMPIIRYKTGDLGCIEDGPCPCGCRLRRLQLRGRMGDQLALNSGNFSPLYLEQQLLQIGGTGNWYQFIVENGELSILV
EASDSSVDRAELLTEIKLRIEASIQQSCDVQVVDYVERDAGKARRVIRL
>lcl|NZ_CP026520.1_prot_WP_042233934.1_435 [locus_tag=PC41400_RS02260] [protein=amidohydrolase family protein] [protein_id=WP_042233934.1] [location=complement(498710..499471)] [gbkey=CDS]
MIIDGHAHISRQDYSSPERLLDDMKSAGIDKAVLVPGGMLDVRRMTRYVTGQEKPILKDPPNDTVLDIIREYPDKFYGIC
CINPNDGAPAALASLEKAYRNGFKGLKMSPLVHQVSLMGDVVKELAALCGDYGIPFYTHLVIQANANTAALGSLAEAFPS
TNFVIGHMGFGPLDIDAIELAARRDNVYLETSTGNPLAIQTAVKQAGPDKLMFGSEFPLSDPFVEKVKIDRLSISDADKE
TILHHTISRLLGL
>lcl|NZ_CP026520.1_prot_WP_042233932.1_436 [locus_tag=PC41400_RS02265] [protein=deoxyribose-phosphate aldolase] [protein_id=WP_042233932.1] [location=complement(499468..500352)] [gbkey=CDS]
MSKLLRLSKIFHPKSGRAFMLPVDHGTTYGPLNGISSMPALLRMANRFRLQAIIAHKGAIRRALESSESPWGTEFILHLS
ASTQMSPDMSNKQIVSTVRHGLQIGAIGLSVHINLGVPQEHEMLRDLGLICDEAYAWGLPVLAMMNVPEEDLSGGAAVTN
KIAHAVRVAGELGADLVKVQTPVTPQRAAEAVAAFNIPVLMAGGSSVGDKLAFFRGIREMLDAGGKGLCIGRNLFEDSRP
DDMCRALRDLVHKDCSAEEAYETYLAASSRMPLIAKQTDHGISAAQFDPWEEYA
>lcl|NZ_CP026520.1_prot_WP_042233930.1_437 [gene=tadA] [locus_tag=PC41400_RS02275] [protein=tRNA adenosine(34) deaminase TadA] [protein_id=WP_042233930.1] [location=complement(501234..501767)] [gbkey=CDS]
METVQFHHDSHDPIQAEHEKWMREAMLEADKAKALMEVPIGAVIVRGGEIIGRGHNLRETSLDPTAHAEIIAIREASEKL
GAWRLLDCKLYVTLEPCPMCAGAIVQSRVPQVIYGTPDPKAGCAGTLMNLLQEERFNHCVDVISGVLQPECASQLTDFFR
ELRRRAKEAKKALPAEE
>lcl|NZ_CP026520.1_prot_WP_042233928.1_438 [gene=alr] [locus_tag=PC41400_RS02280] [protein=alanine racemase] [protein_id=WP_042233928.1] [location=complement(502356..503558)] [gbkey=CDS]
MHPASYRDTWAEISLDAIEHNARTFKKNLQRGCKLMAVVKADGYGHGAYETARAALRAGAEYLAVALLDEALELRAAGIT
EPLLVLGYTPPRSVEKAVRAGVTLTVFTTDVLEAVEAAAAVSSRTPARIHLKVDTGMTRIGVRTAEEARSIARQAIQTPG
VALEGIFTHFATADSADDTFTREQFRRFMNIVDHISDDGIDIPLRHCCNSAATMQYPDMHLDMVRVGISLYGLLPSVASE
RDTHPLKPAMQFKTRLVQVKDVEVGTPVSYGGTFIAERPTRIGTIPAGYADGLSRSLSNRGDMLLHGRRVRIAGRVCMDQ
TMLDITDVENTEEGDPVTIFGKSGDAFLHVDEIAELMDTINYEVVCLIGKRVPRVYLYKGKPAASVNHLIGEIPAQDSPS
>lcl|NZ_CP026520.1_prot_WP_042233926.1_439 [gene=ald] [locus_tag=PC41400_RS02285] [protein=alanine dehydrogenase] [protein_id=WP_042233926.1] [location=complement(503720..504853)] [gbkey=CDS]
MRIGVPKEIKNNENRVAVTPSGVVAFVKAGHEVLVEKNAGIGSGFTNEEYLTAGASLIDQAADVWAGADMIMKVKEPLAS
EYDYFREGLILFTYLHLAAEPELAKALTEKGVTAIAYETVEVNRTLPLLTPMSEVAGRMSAQIGAQFLEKPHGGKGILLA
GVPGVQRGKVTVIGGGVVGTNAAKLAVGLGADVTIIDLSPERLRQLDDLFGNQVNTLMSNPLNIAEAVAASDLVIGAVLI
PGAKAPKLVSEAMIASMKPGSVVVDVAVDQGGIIETVDKITTHDNPTYVKHGVVHYAVANMPGAVPRTSTLALTNVTLPY
ALQIAGKGVLQAVKDNAALQLGVNVAGGSITYEAVARDLGYEYVPVGKALDKISVSS
>lcl|NZ_CP026520.1_prot_WP_042234002.1_440 [locus_tag=PC41400_RS02290] [protein=helix-turn-helix domain-containing protein] [protein_id=WP_042234002.1] [location=complement(504991..506217)] [gbkey=CDS]
MSNELNPFDRSFESLEALADTISEVLGCAITIEDANHRLIAYSTHEAQTDPVRIATIVGRRVPEKVIASLWREGFIQQLM
KSEDPVPIKALDDVKLGRRIAISIRNNREVLGYIWVLEDARLLEEADFLQLKHASRAAKTKLLQLQQQKRKEQEGFEDFF
WQLLTGHISSGDVVRQKSDKLGIPLMDCFQVMIVEFDSDISEQRAQQIRYLISTTQRIRIALHIVDRTQLVLLGTSSSRN
QFAADSSEFLRLFAQQMKDRFGVSPGQTAIGNIYEDYGLVEKSYREALSVLQVKRQFPQETKRFIHYRDLGFYRFLPLMA
EENRIHAHTNPSLLKLRQYDREHNGNLLHTLEIFLLQDSNVKDAADALHVHTNTLNYRLKRISEIGNIDLKDMDQKVSLY
LDLKSEKL
>lcl|NZ_CP026520.1_prot_WP_042233924.1_441 [locus_tag=PC41400_RS02295] [protein=amino acid permease] [protein_id=WP_042233924.1] [location=506568..507995] [gbkey=CDS]
MSKKTQTNELQRGLEQRHITLMSLGAAIGVGLFLGSASAIKLAGPGILLAYALGGLVIFFIMRALGEMAIENPVAGSFSR
YARNHISPLFGYLTGWNYWFLWIITCMAEITAVGIYMEYWFPGTPRWLWALGALVLMSAINLIAVKAYGEFEFWFALIKI
VTIIGMIVIGAGMIFFGLGNGGIAVGFSNLWSHGGFLPNGVTGLLLSFQMVMFAYLGVEMIGVTAGEAQNPEKTLAKAID
TVFWRILIFYIGALVVIMSIYPWDEIGKAGSPFVMTFEKIGIPGAAGIINFVVLTAALSSCNSGIFSTGRMLFNLAQQGE
APDSFRKVSKTGVPGKAIVVSAVALLIGVILNYIVPEKVFAWVTSIATFGAVWTWAMILISQMRYRKKLSPDQVRNLKYK
MPLAPYTSYISLAFLVLVIVLLGFSEDTRVALIVGPIWLIALVVVYYGRGYHRKNALSEKNSDSAGININPQSEL
>lcl|NZ_CP026520.1_prot_WP_042233922.1_442 [locus_tag=PC41400_RS02300] [protein=hypothetical protein] [protein_id=WP_042233922.1] [location=508182..508637] [gbkey=CDS]
MSANREAMGQKQFAKGHFLFILGTCLLSLLLGILYTPIWGGETAFSIQFIAHEIVAEPRGLKLWAYLSIGLGAILLAGLV
GLIRPGKACGLSCTVLAGIAIVLVLIQLPPVFLWGMMLPVDAAAWLPILLHGTALFVGLCAAGLLKNLRNK
>lcl|NZ_CP026520.1_prot_WP_042233919.1_443 [gene=ychF] [locus_tag=PC41400_RS02305] [protein=redox-regulated ATPase YchF] [protein_id=WP_042233919.1] [location=508875..509975] [gbkey=CDS]
MALKAGIVGLPNVGKSTLFNAITQAGAESANYPFCTIDPNVGVVEVPDERLQKLADIVNPNRIVPTAFEFVDIAGLVKGA
SKGEGLGNKFLANIREVDAIVHVVRCFVDENITHVSGKVDPISDIETINLELILADLDSVEKRIDRSRKNMKGGNKQYAA
EVECFEKIKEALYNDQPARSVELTDEERVLVRDLHLLTMKPVLYAANVSEAEVADGDNNPYVQKVREFAAQEGAEVVAIS
AKVESEIAELEGEDKAMFLEELGLQESGLDRLIKAAYKLLGLYTYFTAGVQEVRAWTIRKGTKAPQAAAVIHTDFERGFI
RAEVVSYEDLTATGSMNAAKEKGQLRLEGKEYIVQDGDVMHFRFNV
>lcl|NZ_CP026520.1_prot_WP_042233918.1_444 [gene=prfA] [locus_tag=PC41400_RS02310] [protein=peptide chain release factor 1] [protein_id=WP_042233918.1] [location=510192..511262] [gbkey=CDS]
MLDRLQALADRYEKLSELLCDPDVTSDTKKLREFSKEQSDLQEAYDAYVEYKDVLEQLDAAKAMQAEKLDDEMRDLVKME
IEELSGRKDELEERLTVLMLPKDPNDDKNVIVEIRGAAGGDEAALFAGDLYRMYSKYADAQGWRTEVLEANLNDLGGFKE
IIFMINGKGAYSKMKYESGAHRVQRIPVTESGGRIHTSTSTVGVMPEMEEIDIEIKDSDIRVDTFCSSGAGGQSVNTTKS
AVRVLHVPTGIMATCQDGKSQNDNKAKALQVLRARIYDVKRQEEEAKYAGERKSKVGTGDRSERIRTYNFPQSRVTDHRI
GLTLHKLDTILNGDMGEIVNALTMTAQAELLEKGEQ
>lcl|NZ_CP026520.1_prot_WP_042233916.1_445 [gene=prmC] [locus_tag=PC41400_RS02315] [protein=peptide chain release factor N(5)-glutamine methyltransferase] [protein_id=WP_042233916.1] [location=511270..512175] [gbkey=CDS]
MNRLIYQAGMSIREAYTQASSFFKERQIQDGAICAEILLQHVLGLNRTEFFFRMEEPFPAAHAEQWGEVVERKAAGEPVQ
YITGEQEFYGLTLRVTDAVLIPRPETELLVEEILREGSRLFPQGAPLLADIGTGSGAIPVSLLHARPDWRAAACDLSPEA
LEVAKGNAGQCGVGERISFYEGDLLLPCVERELPVDILVSNPPYIPAGDSPSLQPEVRDYEPHLALFGGEDGLDLYRRLA
GQIPLLPRQPRLVGLEVGIHQAEAVADLLRAAGEWSEIRFVKDLAGIDRHVIAIQDPSEKE
>lcl|NZ_CP026520.1_prot_WP_042233914.1_446 [gene=spoIIR] [locus_tag=PC41400_RS02320] [protein=stage II sporulation protein R] [protein_id=WP_042233914.1] [location=512540..513250] [gbkey=CDS]
MVIRKSYKHLFYTFFALAIMMMSWEGTRSNAALAEGAIPDQSIRLRILANSDKPEDQAIKRLVRDAIVAQMNEWVTGPHS
IDEARQIVQQNLPVLEEVVRNTLTSRGLGGLPYTVELGQVPFPTKMYGNRVYPAGDYEALRVSLGEGKGQNWWCVLFPPL
CFVDGVSGEAVAQAAPAEDGGKGTDEAKTAGVQKKTADAPAEGSAAAATAGGDTEVKFFLWELLQGIIGFFLNLFR
>lcl|NZ_CP026520.1_prot_WP_042233912.1_447 [locus_tag=PC41400_RS02325] [protein=L-threonylcarbamoyladenylate synthase] [protein_id=WP_042233912.1] [location=513280..514365] [gbkey=CDS]
MTTTKLWKIHDEAGTGTGTRETADKAIAEAAGLLVQGETVAFPTETVYGLGADATNSAAVEGIYRAKGRPSDNPLIVHIA
DRSQMDAFAEGTPAEPLRRLMDVFWPGPLTVIVPVRAGAVSPLVTAGLSTVGVRIPDHPVALALLRAAGCPVAAPSANSS
GRPSPTMAEHVADDLGGRIGGIVDGGPTGVGVESTVVEWASGAIYILRPGGVTAEQLREALPDVPVREPARTEEPAGTPR
APGMKYAHYAPKGSMRVVLGDDPKAVAEWMRRDLDAAKARGERTGVLTYDERRGLHHADAVVSSGRLAEPETIARGLYAA
LRQFDREGATYILAEGCPDTGIGRAVMNRLLKAAGQEVIRV
>lcl|NZ_CP026520.1_prot_WP_042233910.1_448 [locus_tag=PC41400_RS02330] [protein=manganese efflux pump] [protein_id=WP_042233910.1] [location=514615..515187] [gbkey=CDS]
MAALAGVSSFVWGQLVTIFIMAIALGFDAFSLGIGIGMRGIRLLHILKISIVIGLFHILMPLLGMYAGHYLGNLLGQVAV
AAGGGLLILLGSHMIYSSFRGEDNPAFDHRTFWGMTLFALSVSVDSFSVGVSLGIFATDVVLTVLLFGLLGGGMSILGLM
LGRHVGSWIGEYGEALGGVILLLFGLRFLL
>lcl|NZ_CP026520.1_prot_WP_042233908.1_449 [locus_tag=PC41400_RS02335] [protein=low molecular weight protein arginine phosphatase] [protein_id=WP_042233908.1] [location=515278..515859] [gbkey=CDS]
MKQILFVCTGNTCRSPLAEGILRHMAQQDNLEVHIRSAGVFASDGSPISRQSSSVLREKGCEASISSSLLRSADVDWADL
ILTMTQDHKRAVIQRFPQAADKTHTLKEFVEDDPEVALAAEEWERLIAEVQVKHALSEEITVEERTRLIHLEKKLPNYDI
SDPFGGSLELYRHTAEEIEEALVKLVQKIKESR
>lcl|NZ_CP026520.1_prot_WP_042233906.1_450 [gene=rpiB] [locus_tag=PC41400_RS02340] [protein=ribose 5-phosphate isomerase B] [protein_id=WP_042233906.1] [location=516207..516665] [gbkey=CDS]
MKIAIGADHAGYRLKDVIVPYLESLGHEVKDVGCNCEQSVDYPDYAVPVCDMVLNGQADKGILICGTGIGMTIAANKMPG
IRCALVHDMFSAKATREHNDTNVLAMGERVIGPGVAQEIVKIWIETEFSHGERHENRLKKVKLLEEKNTIHP
>lcl|NZ_CP026520.1_prot_WP_042233904.1_451 [locus_tag=PC41400_RS02345] [protein=TIGR01440 family protein] [protein_id=WP_042233904.1] [location=516698..517357] [gbkey=CDS]
MNDGEQAWLDGIGPAVEANFRALIEAGGLRPGQIVVFGTSTSEVRGQRIGTSGTPEAAERIFTAAVRVAEEHGLRLAFQC
CEHLNRALVVEEALVEQDSRLEPVNVVPVPKAGGSMAAHAYRHLERACVVEEIAAHAGIDIGDTLIGMHLRRVAVPVRPP
VKWVGEAHVTMAYTRPKLIGGQRAVYEAVPESPDIPVFAGAAAPSRTSPGSSADSDNCE
>lcl|NZ_CP026520.1_prot_WP_042233902.1_452 [locus_tag=PC41400_RS02350] [protein=serine hydroxymethyltransferase] [protein_id=WP_042233902.1] [location=517460..518710] [gbkey=CDS]
MVEFLRKQDPKIVEAMNLELGRQRDKIELIASENFVSQAVLEAMGTVLTNKYAEGYPSKRYYGGCEYVDIVEDIARDRAK
ELFGAEHANVQPHSGAQANMAVYLAALKPGDTVLGMNLAHGGHLTHGSPVNASGILYNFVAYGVSEQDSRIDYDDVRKAA
FKHRPRMIVAGASAYPRIIDFEKLGQIANEVGALFMVDMAHIAGLVAAGLHPNPVPHAHFVTTTTHKTLRGPRGGLILCR
KPWAAAIDKAVFPGSQGGPLMHIIAAKAVALGEALQADFKTYGKNVIDNAKALAEGLTADGINLVSGGTDNHLILIDLRS
LNITGKDAEHLLDEVGVTVNKNAIPFDPTSPFVTSGIRIGTPAATSRGMDTEAMKQIAKIIAMTLKNPSDSAVHEQARGL
VKEVTSKFPLYKGLEY
>lcl|NZ_CP026520.1_prot_WP_009676968.1_453 [gene=upp] [locus_tag=PC41400_RS02355] [protein=uracil phosphoribosyltransferase] [protein_id=WP_009676968.1] [location=518950..519579] [gbkey=CDS]
MSKVFICDHPLIQHKLTYIRDVNTKTKDFRELVDEVATLMAYEITRNIQLENIQVKTPVATADCRIISGRMLGIIPILRA
GLGMVDGILNLIPAAKVGHIGLYRDPETLQPVEYYVKLPTDVQERELIVVDPMLATGGSANLAITALKNRGCVQIKLMCL
IAAPEGIEAVQKEHPDIDIYVAAIDDYLNDHGYIIPGLGDAGDRLFGTK
>lcl|NZ_CP026520.1_prot_WP_042233900.1_454 [gene=wecB] [locus_tag=PC41400_RS02360] [protein=UDP-N-acetylglucosamine 2-epimerase (non-hydrolyzing)] [protein_id=WP_042233900.1] [location=519604..520740] [gbkey=CDS]
MKRLKVITIFGTRPEAIKMAPLILELQKHPEQIESLVAVTAQHRQMLDQVLELFNIQPDFDLNVMKDRQTLNEVTIRVLQ
GLEPVFREAKPDLILVHGDTLTTFLASYSAFLQQIKVGHVEAGLRTWNKMSPYPEEMNRQLTGVLADLHFAPTNWSAGNL
RKENKSEDSIYVTGNTVTDVFQYTVKKDFHHPVLDWAEGKRLILMTAHRREAQGEPHRQIFNAVKRIADEFDDVAIVYPV
HPSPAVKEPAYEILGNHPRIKLIDPLDVFEFHNFYPHAYMILTDSGGLQEEAPSFGVPVLVLRDTTERPEGVEAGTLELV
GTDGQKVYERARALLTDPQTYDRMKGAANPYGDGQASRRIVQGILHHFGYKTEKPDSF
>lcl|NZ_CP026520.1_prot_WP_042233898.1_455 [locus_tag=PC41400_RS02365] [protein=AtpZ/AtpI family protein] [protein_id=WP_042233898.1] [location=520938..521171] [gbkey=CDS]
MKNKGNRDNPWRAAGLVGALGLDIAICTMLGYFAANFAVQRFGGPRSWIVAGVMLGLFAGIASVIMLLKKFLEEPDE
>lcl|NZ_CP026520.1_prot_WP_042233896.1_456 [locus_tag=PC41400_RS02370] [protein=ATP synthase subunit I] [protein_id=WP_042233896.1] [location=521164..521532] [gbkey=CDS]
MNDLLKKAMRIALLLLAACLLVWALVPSVRTVAAGLLVGAIASGINALLLQRRVDLIGKTVAERGAQRMSLGFLSRLATV
LLAVMIALKFPQHLDIASTIIGCFLVPFILLLVGILSRPHKP
>lcl|NZ_CP026520.1_prot_WP_042233894.1_457 [gene=atpB] [locus_tag=PC41400_RS02375] [protein=F0F1 ATP synthase subunit A] [protein_id=WP_042233894.1] [location=521559..522338] [gbkey=CDS]
MHEFPVLHFENGIDIDISTFLAIIVSCVVVLVLALLSVRNLSVDKPSKLQNFMEWVVEFVHNTIASTMPLSKAKSYISLG
LTLIMFIFVSNLLGLPFGIVTDHTQGFTLFGQQVLTDTRIHELEEAGKHASVAWWKSPTADLSVTSALALMVFVMTNTLG
LTQNTKHYLKHYFEPFWFFLPLNIIKELSKPLTLALRLFANIFAGEVLISTIMMLGVVGIPLLAVWQGFSIFVGAIQAFL
FTILTMVYISQATVHHEDH
>lcl|NZ_CP026520.1_prot_WP_009677035.1_458 [gene=atpE] [locus_tag=PC41400_RS02380] [protein=F0F1 ATP synthase subunit C] [protein_id=WP_009677035.1] [location=522445..522657] [gbkey=CDS]
MEFLAAALAIGLGALGAGIGNGLIVSKTVEGIARQPELRGTLQTTMFIGVGIVEVVPIIGVVLGFLIFFK
>lcl|NZ_CP026520.1_prot_WP_042233891.1_459 [gene=atpF] [locus_tag=PC41400_RS02385] [protein=F0F1 ATP synthase subunit B] [protein_id=WP_042233891.1] [location=522768..523253] [gbkey=CDS]
MSWNWESFVFALVAFGILYWLLSKYAFGPLLATMEERRKLIADQMNSADASRVQAEQMLEEQKQVIQEARKEAYQMIEQA
KLTSSKQAEEIVDKAKSEADRVKEEAVREIESEKNKAVAALRGQVSAMSVMIASKIIEKQVDEKSQEELVNHYLNEVGGS
K
>lcl|NZ_CP026520.1_prot_WP_042233889.1_460 [locus_tag=PC41400_RS02390] [protein=F0F1 ATP synthase subunit delta] [protein_id=WP_042233889.1] [location=523250..523798] [gbkey=CDS]
MSDIVVAKRYAKALFEAAKEQNAVAQVETELTEIVQTIQDHAELRSVLDHPNIDVPAKRQLLSNIFADRISPLVMNTIKL
LIDRRRQSLLPIVLSDFVQISNESLGQADATIYTALPLSEVQVKEIADNFSRMTGKQIRVQTVVDPSLLGGITVRIGDRL
YDGSLAGKLERLEKSLKQAQAM
>lcl|NZ_CP026520.1_prot_WP_042233887.1_461 [gene=atpA] [locus_tag=PC41400_RS02395] [protein=F0F1 ATP synthase subunit alpha] [protein_id=WP_042233887.1] [location=523815..525329] [gbkey=CDS]
MSIRPDEISTLIKQQIENYTADLEVVEVGTVITIGDGIARVHGLENVMAGELLEFANGVVGMAQNLEENNVGIIILGPYS
DIREGDQVKRTGRIMQVPVGEALLGRVVNALGMPIDGKGPIEASGYRPVESAAPGVMARKSVHEPMQTGIKAIDALVPIG
RGQRELIIGDRQTGKTAIAIDTIINQKGNGVKCIYVAIGQKQSTVSNVVDTLRRNGALDYTIVVTASASEPAPLLFLAPY
SGCAMGEYFMYKGEHVLIVYDDLSKQAAAYRELSLLLRRPPGREAYPGDVFYLHSRLLERAAKLNDELGGGSITALPFIE
TQASDVSAYIPTNVISITDGQIFLESDLFYSGQRPAINVGISVSRVGSSAQVKAMKKVAGTLKTDLAQYRELAAFSQFGS
DLDKATLSRLNRGERTLEILKQGINEPMPFEKQVISIYAATKGFLDELPVKDVTRYEKEFLAYLDANKPEIASSIRDTKD
LTSDTEKALVAAIETFLKGFSVSE
>lcl|NZ_CP026520.1_prot_WP_009676985.1_462 [gene=atpG] [locus_tag=PC41400_RS02400] [protein=ATP synthase F1 subunit gamma] [protein_id=WP_009676985.1] [location=525378..526241] [gbkey=CDS]
MAKGIREIKRQIKSKQNTRQITKAMEMVAASKLKRAQNNAEASRPYADKIREVVFSVAAGSKGVQHPMLQVREVKRTGYL
VLTSDRGLAGGFNANVLRKALNAIRERHKSTDEYAVFVVGRKGKDFFKRRNIPVFDEVSGLSDSPTFADIKQLASSAVQN
FIDGNYDELYVYYNQFHNAITQTPVEKKLLPLEQPESGSSAVSSYEYEPSAEGVLEVLLPKYAETVIFSAVLESKASEFG
ARMTAMGSATKNATKMIAELTLSYNRARQAAITQEISEIVAGANALN
>lcl|NZ_CP026520.1_prot_WP_042233884.1_463 [gene=atpD] [locus_tag=PC41400_RS02405] [protein=F0F1 ATP synthase subunit beta] [protein_id=WP_042233884.1] [location=526286..527695] [gbkey=CDS]
MSKGRVVQVLGPVVDIEFERGHLPEILNAIKIQKTAQSADQENSEVTVEVAVHLGDNQVRCVAMSSTDGLVRGMAAIDSG
AAISVPVGPATLGRVYNVLGDPIDGKGPADRTLTSPIHKQAPEYVNLTTQAEMLETGIKVIDLLAPYMKGGKIGLFGGAG
VGKTVTMQELIHNIAQEHGGISVFAGVGERTREGNDLYHEMSDSGVIAKTAMVFGQMNEPPGARLRVALSGLTMAEYFRD
EEGRDVLLFIDNIFRFTQAGSEVSALLGRMPSAVGYQPTLATEMGQLQERITSTTKGSVTSIQAIYVPADDYTDPAPATA
FAHLDATTNLERAIAAKGIFPAVDPLASSSRILTPQVVGEEHYEVAQGVKRLLQRYKELQDIIAILGMDELSEEDRIAVN
RARRVERFLSQPLHVAEAFNGMPGLYCPVKETVRSFRELLDGKHDDLPEAAFVNVGSIDEAVEKARTMV
>lcl|NZ_CP026520.1_prot_WP_042233882.1_464 [locus_tag=PC41400_RS02410] [protein=F0F1 ATP synthase subunit epsilon] [protein_id=WP_042233882.1] [location=527843..528247] [gbkey=CDS]
MSTFLLEIVTPERIVYTQNVHMVSVEGIEGQLGILPNHIPFVTPLKIAPATIKHENGKQETVALNGGFMEVRKDKVVILA
ESAELPSDIDIERAKSAKERAEKRLTGKKDEADFQRAELALKRALNRIGVYEHK
>lcl|NZ_CP026520.1_prot_WP_009677054.1_465 [locus_tag=PC41400_RS02415] [protein=NADH-quinone oxidoreductase subunit A] [protein_id=WP_009677054.1] [location=528438..528803] [gbkey=CDS]
MYAVKYVIVAIFVALGILLPVAALTIGRWLRPNKPNEDKATTYESGNEPVGEGQVRFNIRYYLFALMFVIFDVETVFLYP
WAVAYNKLGLFALVEMCIFVLLLGVGLLYAWKKKVLQWTSI
>lcl|NZ_CP026520.1_prot_WP_009676983.1_466 [locus_tag=PC41400_RS02420] [protein=NADH-quinone oxidoreductase subunit B] [protein_id=WP_009676983.1] [location=528788..529306] [gbkey=CDS]
MDFNLESITPEDRAELDRNVFMTTLEQVKAWARSNSLWPLTFGLACCAIEMMGTGASHYDLDRFGVIFRTSPRQSDVMIV
AGTVTKKMAPLLRRLYDQMPEPKWVIAMGSCATAGGPYVKSYAVVKGVDQVVPVDVYIPGCPPNPAALIYGINKLQEKIR
YEAKTGKQVTNL
>lcl|NZ_CP026520.1_prot_WP_275938051.1_467 [locus_tag=PC41400_RS30525] [protein=NADH-quinone oxidoreductase subunit C] [protein_id=WP_275938051.1] [location=529456..530268] [gbkey=CDS]
MSTGSAGDDIAPPAGPSEQSASAAGTGESAPETPASGAAQRPDRAEARSAATAEAGGTAPAEANGAAPTEGAAAEDGSDK
AKAAAEARAARAAARAAKTAAPAEDAPPKPPSPNQPMLDRLVEILKGEVAGDAVDEAFINEIDGHRPFVVIRGEHWVRAA
EVFLQHEELQCGYLRSVSGVDLGNQLEVVYHLLSIPLRKEYGVRVRTEREAPSVPSVASVWPTADWNEREIYDLLGIDFP
GHPNLTRIMMPDEWVGHPLRKDYEAYDPEV
>lcl|NZ_CP026520.1_prot_WP_042233875.1_468 [locus_tag=PC41400_RS02430] [protein=NADH-quinone oxidoreductase subunit D] [protein_id=WP_042233875.1] [location=530273..531370] [gbkey=CDS]
MLRTEELLLNVGPQHPSTHGVFRIIVKLDGEIIREAIPVMGYLHRGTEKLAENLNYTQIIPYTDRMDYVSAMTNNYVLVN
AVERMLGLQIPERAEYLRILVMELQRIASHLVWWGTYLLDIGAMSPFLYAFRDREMIIDFFNELCGARLTYNYMRVGGVK
WDAPEGWLDKIRAFIPYMKSKLEEYDRLVSGNEIFLARIKGVGRYDADTAIAYGLSGANLRCTGVKWDLRKDQPYSIYDR
FDFDVPVGTNGDCYDRYLIRLEEIVQSLRIIEQALDQIPEGDTMGKAPRVIKPPEGDIYAPIESPRGEIGCFIMSKGRPE
PFRLKFRRPSFVNLQILPKLLVGETMTNLITILGGIDIVVGEVDG
>lcl|NZ_CP026520.1_prot_WP_042233871.1_469 [gene=nuoH] [locus_tag=PC41400_RS02435] [protein=NADH-quinone oxidoreductase subunit NuoH] [protein_id=WP_042233871.1] [location=531370..532377] [gbkey=CDS]
MLDLVGQPLTWTTGLIFAAYGVVLLLIILGFVTYAIYFERKVIGWMQGRYGPNRVGPLGLLQTVADVFKLLIKEDTIPRK
ADRELFILAPVLTFIPAFVVVATIPFTANLIHADLDVGLLYYVALSGISTLGMILGGWASNNKYALLGGMRSAAQMISYE
VPLVISVIGVIMMTGTLNLHKIVEAQGDNFWNWNFLPQIVGFIVFVIAGVSELNRTPFDLPEAESELVAGYHVEYSGFRF
AFFMLAEYVYVFAIASLTTVLFLGGWHAPFAFLAFIPGIVWFLLKFSFIVFFFFWIRATLPRIRIDQLMGLGWKVLLPLA
LLNIFITALYMGLTG
>lcl|NZ_CP026520.1_prot_WP_042233867.1_470 [gene=nuoI] [locus_tag=PC41400_RS02440] [protein=NADH-quinone oxidoreductase subunit NuoI] [protein_id=WP_042233867.1] [location=532424..532876] [gbkey=CDS]
MKGLIKGLGVTLKTMTQKKVTYAYPDVPLQMPDRFRGIQHFEPDKCIVCNMCAKVCPTECITLTGKPNPDPDKKGKIIDT
YDINFEICILCDLCTEVCPTEAIVMTNHFELSAYSRDELFKDMKWLTDNNTLVRKENNSALPKGGAKKDV
>lcl|NZ_CP026520.1_prot_WP_042233864.1_471 [locus_tag=PC41400_RS02445] [protein=NADH-quinone oxidoreductase subunit J] [protein_id=WP_042233864.1] [location=532869..533399] [gbkey=CDS]
MFSGLGSFLSQGENWWFLLFAICAIGGAIYMITFTKVVHMVLSVAFTFVSLAGLYVLLEAEFVGFVQILIYAGAISILMI
FGIMMTNHRAEEKAKSGPVQKILLGVGSLGLFGILFYAIQNTTFPSAAGSLQTGQDNTLEIGKLLYNEYVIPFEIMSVLL
TVAFIGAIVVAKREED
>lcl|NZ_CP026520.1_prot_WP_042233861.1_472 [gene=nuoK] [locus_tag=PC41400_RS02450] [protein=NADH-quinone oxidoreductase subunit NuoK] [protein_id=WP_042233861.1] [location=533401..533712] [gbkey=CDS]
MTSVVTPYLTLAAILFCIGLCGALSKRNGVIVLLSIELMLNAVNLNLVAFAKMGAQPALTGQVFSLFTITVAAAEAAVGV
AILIALYRNKGTGNVDEMNSMKH
>lcl|NZ_CP026520.1_prot_WP_042233859.1_473 [gene=nuoL] [locus_tag=PC41400_RS02455] [protein=NADH-quinone oxidoreductase subunit L] [protein_id=WP_042233859.1] [location=533794..535668] [gbkey=CDS]
MISVYSQYVWLIPLFPLLAYVLITASGRSARRTGLYLSTAAIFASFVVALLILIERLGKGVQDYTWNGFQWLRVGDFAVS
LGFEVNNLNAIMLIVVTLVSFLVHIYSIGYMKDDERVHVFFSYIALFSFAMLGLVISVNLVEFYVFWELVGVCSFLLIGF
WYTKPEAKAAAKKAFIVTRIGDFGLFIGILLLFWSMPGHALDFNSVQNAFTSGQVEGSAATWIALLFFVGAMGKSGQFPL
HTWLPDAMEGPTPISALIHAATMVAAGVYLVARTFDVFLASDTAMQIVAYVGGFTAIFAATIGTAQNDIKRILAYSTVSQ
LGYMMMALGVSATYTTGIFHLFTHAFFKALLFLGAGSIIHAVHTQDITKMGGIGGRMKITAWTFAIGTLALSGIPPFSGF
WSKDAILSQAYETNSLLFWVGIVGAFFTALYMARLFFLVFWGKSRSGQEVHESPASMTFPLLLLAVPAALAGFLYTPWSG
WFGDWLTGRTEAEAVHAGVMLLSTAAGVLGIVLGYLLYARKPGSFVFSAKFPWLYNLVRRKYYIDEIYRTIIVLPLRGIG
YLLQAVDTFIVDGVVRLAAAAAVGLGRTGSRLQNGQVQTYGLITLLGLLVLILAVAGRRFIHVG
>lcl|NZ_CP026520.1_prot_WP_042233856.1_474 [locus_tag=PC41400_RS02460] [protein=NADH-quinone oxidoreductase subunit M] [protein_id=WP_042233856.1] [location=535658..537286] [gbkey=CDS]
MSANLLSLIVFSPLLGILVLLFLPKENKKALQAVCVAATGIPLLLSAWLYAAYSNGVSGMQFKEQLDWIRIPLNKEGVEG
LASFYWTVQYHLAVDGLSMPLVFLTALVFTMAALASIYIKKRWKAYSIWLLLLEVGVFGVFTSQDLFLFFLFFEVTLVPM
FFLIGIWGRVDREKAASRFLIYNGLGSAVMLVAFVILVTTAGISQQIAGNQLTVFYSGDIQTIVHNLLQDPTALVNQVSP
NPQAENPFFLSEGMKTVLLVMLLIAFGIKLPIFPFHTWMLKVHTEAPPPVVMIHSGILLKMGAYGLIRFGILLFPAQAQD
WAWVLALLGVINILYGALLAFVQNEFRLVLAYSSISHMGIVLLGLAAFNPIGMQGAIFQLVSHGLISALMFLLVGSIYER
TSTTELGKLGGLAKSMPFLSAILLVGGLASLGLPGLSGFVSELLAFIGLFGTARVLTIIGAVGIILTAAYVLRGVMKMTY
GPLPERFRELPDARFIEAVPMIALVAFILLLGVYPGILSEPLQQTVNGFHQTIHSLAGKIGG
>lcl|NZ_CP026520.1_prot_WP_042233853.1_475 [locus_tag=PC41400_RS02465] [protein=NADH-quinone oxidoreductase subunit N] [protein_id=WP_042233853.1] [location=537290..538804] [gbkey=CDS]
MEPIRIHVQDLTHLLPEMTLVITAILLSVLDLFLLKGVNRKWIGWLSLAGIAVSIAFVIAQLNPDQAIVLLNESYRVDDF
GNLLKLAFLGGTALIILMSLGSVEDGEIPHVGEYYYFLLPAVLGAMVMASSADLITLFVGLELLSITSYVLVAMKKKNTR
SNEGAFKYVILGGISSAVILYGMSFLYGIAGTTNLGAIQAAIVSSYESYKPLLYVSFFLLLAGFGFKIAAAPFHSWAPDV
YQGAPTPVTAFLAVISKGAAFAILFRVFYVLLALGQFGDEGLQKDIFLALSVVAATAMIWGNTVALRQKNVKRLLAYSGI
ANAGYLLVPLAVHINLVQVDNFSKFFFYLIAYILMNIGAFTVLLIIEKSTGDEETKGFAGLYYRAPWTSAAAVVLLLSLA
GIPVSAGFFGKLYLLFGTMQAHLYWLGAVMMITSVISFYYYFGIIRQMFMRADLEERPVSVPVPLGITLWLCAGLTLLLG
LVPGYLLAYIDRIFSLTRDLLILS
>lcl|NZ_CP026520.1_prot_WP_042233850.1_476 [locus_tag=PC41400_RS02470] [protein=S8 family serine peptidase] [protein_id=WP_042233850.1] [location=538942..541494] [gbkey=CDS]
MNRKFRYKWMYGWLGVVLLVWTVGISNVQAYPVSNDPLRVKQTYLDDIHMDEGWAQAGDNDEPVIVAVVDTGVDLTHPDL
EGRLVPGANLLNPQEKPQDDNGHGTNVAGIIAAKMNNDKGIAGIAPNAKIMPIKALEADGTGGETKLGEGIRYAVDHGAK
IVVLSLGLNKYSSFMAGTVRYAEERGVLLVAASGNEGGSVKYPAAYPTVLGVGGMTSARLADDRSNYGPDLDLVAPWHVY
TTAKGGGYESKEGTSMAAPQVAGAAALLWGRAPYLEPWQLRELIKQTAEDLESPGWDATTGFGLLRVDRLMKEPLLSDRF
EPNDDSGHAPMLPVHTLIHAEFSSGKDTDWFTMDSPYDGFVKLVLKTSDASPVSIRYFRDNKEAEEVFTTDPQNPVVTLP
VQKGKSKIAASLLDPGKAGAVLYTLQTDFQIYQDDFEDNDHQYKATSLPARSQTVKGTFHQENDQDWFMISFKKSGTLRL
KTSVDTARIDPVVLIQKKGEKSTTFDQFGEGATEATPLMNVTPGDYYIRISNVKDNTQPVMGEYTLSVDYTPKLIDPNEP
NNKPYQATTLTLHEDHPGVIESASDLDWFQFRLEQESLVQLKLGSLQRGCDITFTLYDHTLKELDSKSSEGSAENLVLDR
RLPAGLYYIKLQSDRAATQQMYQLNAAAQPLVAGFSDIHGHWAEKAIADLASRHVIDGDGSYKFYPDLPITRAQAAVMLA
NGFRLDKNRPVDYPDLSSSHWAYDSIARTTMAGYMDGYEDGHFRPDQPLTRMEMTQVIANTMQMAGKYRGRSPFLDVSDR
YWGVGILKQMSGDGWIRGFPDGTYRPDDQATRAEFAALLSRFSNVFGRKR
>lcl|NZ_CP026520.1_prot_WP_042233847.1_477 [locus_tag=PC41400_RS02475] [protein=DUF1146 family protein] [protein_id=WP_042233847.1] [location=541499..541750] [gbkey=CDS]
MSNMEELTTVVPQMPLINIVVYLVAVGLSWWALQQFRFDVLLKQPRSPAAKMLQVFMSIGLGYLVASFFLKYLEWSVGLN
GIF
>lcl|NZ_CP026520.1_prot_WP_042233845.1_478 [gene=murA] [locus_tag=PC41400_RS02480] [protein=UDP-N-acetylglucosamine 1-carboxyvinyltransferase] [protein_id=WP_042233845.1] [location=541916..543238] [gbkey=CDS]
MSKIIVRGGRKLAGKIKVNGAKNAVLPIIAASILAAEGESVIHDAPPLDDVLVINKVLRSLGLQLSYHNHSVKVNAQAIE
TCEASYELVRKMRASFLVMGPLLARLGKARISLPGGCAIGTRPIDQHLKGFEAMGAEIELGQGYIEARVEKRLKGAKIYL
DVASVGATENIMMAATLAEGTTIIENAAKEPEIVDLANYLNAMGAVVRGAGTGVIRIEGVEKMKGAVHSVIPDRIEAGTY
MIAAAITRSELYVEGAIGDHLRPVISKLQEMGVHVEEDENGILVCADKPLRAADIKTLPYPGFPTDMQSQMMALQLVADG
TSLVTETVFENRFMHVDEFMLMNASVKVDGRTAVVSGNNKLKGAKVCATDLRAGAALILAALVAEGETEITGLHHIDRGY
VDITSALRSVGADIQRVVPQEEEQEQPQEIGLFAVQPTWA
>lcl|NZ_CP026520.1_prot_WP_042233843.1_479 [gene=spoIID] [locus_tag=PC41400_RS02485] [protein=stage II sporulation protein D] [protein_id=WP_042233843.1] [location=543489..544529] [gbkey=CDS]
MKFKRRSKLKQPALLWTGIAVSGLLLVTVLLPAVVVKQQTGTFADLHGTEAVPSSADMVMIPVYLSKQKQIQTVSLEQYV
KGVVAAEMPAEFSEEALKAQAIAARTYIVRRILTGDFSNVPVKEAWVTDTIQHQAYLTDDQLQKQWTGKAYESNSKKIDT
AVKATADFILTYEGKPIQATFFSTSNGYTEDVRNVWGSDLPYLKSVSSPWDEKLSPKFKQTITLSYKDILQKLGLPASSS
LNSGSIKVLERTESKRVKSAMIAGKTFTGKEIREKLGLNSTDFEADLSGTDWTFTVKGYGHGVGMSQWGANGMAKEGKKA
AEIAAYYYPGAKLTSASTLLNKKTQK
>lcl|NZ_CP026520.1_prot_WP_042233841.1_480 [locus_tag=PC41400_RS02490] [protein=M23 family metallopeptidase] [protein_id=WP_042233841.1] [location=544607..545353] [gbkey=CDS]
MRDGKQEFHNKEEAPQTVTGAANVPNSAWKRFLAKKWVFPATYVAAAAIILTLMWVYQDAVKKPVATNTPVNSVQSDKPA
GETANKDNAVPVGAKAESMQWPVKDRSQVTISLPSYDSKGSNEAKQAAMIEYGDTITPHVGVDLVRADKQAFDVSAALSG
KVTAVDKNPVVGNLVEITHDNGLVTVYQSLGEVKVAKGAAVKQGDVIAQAGRNELEKDQGVHLHFEVRKAEDGTAELPEQ
LLPEFKPQ
>lcl|NZ_CP026520.1_prot_WP_009676986.1_481 [gene=spoIIID] [locus_tag=PC41400_RS02495] [protein=sporulation transcriptional regulator SpoIIID] [protein_id=WP_009676986.1] [location=545552..545842] [gbkey=CDS]
MHDYIKERTIKIGTCIVETRNTVRTIAKEFGVSKSTVHKDLTERLPEINPELANQVKTILEYHKSIRHLRGGEATKIKYR
KTKPVKSAEPLVTVKS
>lcl|NZ_CP026520.1_prot_WP_042233839.1_482 [locus_tag=PC41400_RS02500] [protein=rod shape-determining protein] [protein_id=WP_042233839.1] [location=546010..547011] [gbkey=CDS]
MLSKDIGIDLGTANVLIHVKGRGVVLDEPSVVAIESDTKRVLAVGEEARRMVGRTPGNIVAIRPLREGVIADFEITEAML
KHFISKVGGKSWYSHPRILICAPTNITSVEQKAIREAAERSGAREVFMEEEPKAAAIGAGMDIYQPSGNMVVDIGGGTTD
VAVLSMGDVVTSSSIKIAGDKFDTSITKYIKNKYKLLIGERTSEDIKVQIGTVQADSVQNEMDIRGRDMVSGLPLTITIR
SSEVQEALWDPISSIVTAAKSVLERTPPELSADIIDRGVILTGGGAMLHGLDQLLADELKVPVLIAEDPMHCVVKGTGIM
LDNLDKVTKRKFV
>lcl|NZ_CP026520.1_prot_WP_042233838.1_483 [locus_tag=PC41400_RS02505] [protein=flagellar hook-basal body protein] [protein_id=WP_042233838.1] [location=547021..547839] [gbkey=CDS]
MLRGLYTAASGLTMQQRKHDTVTNNIANLNTPGFKQETTSARSFPEVLISRIRGGEAGGGTVVNVGKMNTGVFAEETARL
MVQGPLQQTGNPLDYALVSDIQVNGLKFDNGGKAVGPDGTVVYQPEALFVVENPQVAGERHYTRNGKFTVDAEGFVANAD
GYRLVAANGQTVRYDPANPPIPLVMKVQNPNVLLPEGNGLFTVGANDEGQVVALAAGDQVEVRKGFVERSNVDPTQAMVD
MTTAARAYEANQKMVQFYDRSMDKLVNDVGRV
>lcl|NZ_CP026520.1_prot_WP_042234000.1_484 [locus_tag=PC41400_RS02510] [protein=flagellar hook-basal body protein] [protein_id=WP_042234000.1] [location=547915..548784] [gbkey=CDS]
MNNSIINASVTMQGLQQKLDVLANNMANVNTVGYKKKEASFENVLTNTMNQPQGFMKTGRMTSLGLPQGWGAKLVGLQNI
MTQGSLKPTDEPYDLAIEGDGLFEVTIPGSNTPAWTRDGALKLSAYNKVDPATGTVTKAMGLATKDGYFVKDTASNQPIE
IAIDPADPTLQNSRVRVDEQGRVVVYNEANPASTGRQVGQIKLVKVINPQLLGEADGNYFRLPANVAQNNVFDVNTNGVT
IKQGFLEQSNVDLQQEMSELVMVQRAFQLNSRAISSADTMMGLANNLRG
>lcl|NZ_CP026520.1_prot_WP_241688876.1_485 [locus_tag=PC41400_RS02515] [protein=DNA-directed RNA polymerase subunit beta] [protein_id=WP_241688876.1] [location=548814..549071] [gbkey=CDS]
MEEESKETEAASSTQAKEPGTGKPGGASRSAGALRVLKKLRVPLIFLGALIVGLWIGYSGIGGQPTLDIFRISTWKHLFD
LVFAD
>lcl|NZ_CP026520.1_prot_WP_042233833.1_486 [locus_tag=PC41400_RS02520] [protein=CDP-alcohol phosphatidyltransferase family protein] [protein_id=WP_042233833.1] [location=549188..549742] [gbkey=CDS]
MNLPNLLTISRFFLIPVYLYIFYAGYTKLAFLVVVLAGITDIADGYIARKRNLVTQMGIMLDPLADKLMMIAVIISLLSS
GMIPWLAAAAFFIRDIGMIAGSAFFHFRGKQTLPANAMGKLTTVLYYLAVLLILFHVPYATEYLAAVILFSFVTSIIYVA
QFNSVNRKTRRLSTEETGPVKKGS
>lcl|NZ_CP026520.1_prot_WP_009677001.1_487 [gene=fabZ] [locus_tag=PC41400_RS02525] [protein=3-hydroxyacyl-ACP dehydratase FabZ] [protein_id=WP_009677001.1] [location=549944..550372] [gbkey=CDS]
MLDNKQIQEIIPHRYPFLLVDRILEVEDGVRAVGLKNVTANEPFFQGHFPGYPVMPGVLILEALAQVGAFAILNVEANRG
KIGLLAGVDGFRFRRQVTPGDTLTLEVKITRLKGTIGKGQATAKVGEELVAEGEIMFALANA
>lcl|NZ_CP026520.1_prot_WP_042233832.1_488 [locus_tag=PC41400_RS02530] [protein=phospho-sugar mutase] [protein_id=WP_042233832.1] [location=550419..552113] [gbkey=CDS]
MTRVQNLYEKWLQDPSIDEETKKELQSIAGQDKEIEDRFYKDLEFGTGGLRGVIGAGTNRMNLYTVGKATQALAEFALKN
GGAHPSVVIAYDSRHQSPEFALEAALVLAGNGVKAYVFETLRPTPELSFAVRELKASAGIVVTASHNPPEYNGYKVYGAD
GGQLVPHDAEQVLANIQKIEGFDSVRKISRTEAEEKGLLEWLGGEMDEKYYTAVTSVSQHPDVIKQTSGDFRIVYTPLHG
TGNKPVREALKRIGFEQVRVVKEQEQPDSNFSTVKSPNPEEREAFTLALRDAKEWDADIIVGTDPDADRMGAVVKDDKGE
YFVLTGNQSGAIIVHYLLGSLKERGELPSNGVVIKTIVTSEMGAVIAESYGAKVLNTLTGFKYIGEKMTEFESTGEHQFL
FGYEESYGYLAGTYARDKDAVVAAMLICEAAAYYKSQGKTLYDVLQELYRQHGFFLEKLESKTLKGKDGVEQIGRMMEDW
RNNPPASIEGVEVAEVEDYSRGINGLPKENVLKFKLKDGSWFCLRPSGTEPKIKFYFAVKGTTGDEAANQLQGLVRFVLD
RVEQ
>lcl|NZ_CP026520.1_prot_WP_084706674.1_489 [locus_tag=PC41400_RS02535] [protein=DUF5693 family protein] [protein_id=WP_084706674.1] [location=552243..554276] [gbkey=CDS]
MLRKLLWWLVIIGLLASLPLAFFREQTEKSANQVEFVFDYRDLLEMADVRPNPQGFVSEQLQAMKKAGIGSLAVYESTLS
ELRLSRRIEVFSSKDATSLTQKPISPNENFTYILFTDPASESQIKPMIETAFNKLGVKTSPWSFKNQAGMIIEMNMEEAN
MKPMDPDPITLQMMKDEGFRIVVRLGNKREFVTNEMDALLAKLYKLGARRIIIDGDYAPGYSDDPEQVTLHKMGILLKKY
DMGLANIEMLKQPQKGFGSLAKDTDYNVVRLHSFTEKDGEKLTENLTKEDQEARIQQTADRFVLAVKDRNIRMLFLNGRA
VKNLDKAAILTPLDSLYESLNGKDGAIKRIEDAGFKVGPAKQFDYTYNTYQNVARFFTLIGGVSLIVLLVGYFIPEAVLA
LFIIGLLGAAGLFFKSPDLYAQGLALACAISAPSIAVIEAIRSIQSGRAAKAKSAVLYALGLLIKTTLISLIGVVYLVGL
LNHISYFLVLQQFRGVSLLHLLPIALVGLYLLFFRETLNYRERGERVKKLLSTNISVLWIIAAAAAAGVIYYYLSRTGNE
GQTSAIERYLRSFLENTLGVRPRNKEFMLAHPLFLLGAYLSVKYRNAVYLLLIGVVGQLSIVDTFAHLHTPLEISGIRIT
YGIVFGAMIGLVLIAVWEIASRSWKAWVAKSRTRVNG
>lcl|NZ_CP026520.1_prot_WP_042233827.1_490 [gene=csaB] [locus_tag=PC41400_RS02540] [protein=polysaccharide pyruvyl transferase CsaB] [protein_id=WP_042233827.1] [location=554240..555385] [gbkey=CDS]
MGREIAHKSKRLVVSGYYGFDNSGDEAVLQSILFALKEQGEAAGISIEPVVLSANPERTREMYGVEACHRMKVPDLFRAI
KSSDGLISGGGSLLQDATSSRTIPYYLLVLKLAQWMRKPTFIYSQGIGPVNNPRFFSWIRSTFKKCAMITVRDQESADLI
GRMGVPASSVSVVPDPVMGLPLRGEAASGKAGAADASGETGRVPVVGVSVRFWNADRSELGALAEALRAVMDSRRIEVRL
LPFHLPSDEEASAYVAERLSPAGEGGDVRIVRGVTHPQDMLAEVSACDVLVGMRLHSLIYAASQFVPMVGVSYDPKIDQF
LARLGMKAAASTQSFDAAAVAAEIGRLLDGRREWADGKKAAIDRLKKEAQLPAQQIMQFYR
>lcl|NZ_CP026520.1_prot_WP_042233826.1_491 [locus_tag=PC41400_RS02545] [protein=WecB/TagA/CpsF family glycosyltransferase] [protein_id=WP_042233826.1] [location=555401..556174] [gbkey=CDS]
MTANANASVPIVRIYGVPVSKMNMKDTVAYLTGAIESRIPHQIITANPIMIMSALEDPAYMEMMRRVELIVPDGAGLVWA
AGYVGQPVAERVPGIDLIHELMRTGESRRWKVYLLGTTPDTIQAAAAKLARDYPLIDIVGFRDGYFGEAEDRQVVDSIRQ
LEPDILLVGRSSSTQEPWISKYKRDLDVPVIMGVGGSFDVLAGKLKRAPKIFQKLKLEWFYRLLQEPSRFGRMLALPKFA
WKVIRDKQNVLKTDENH
>lcl|NZ_CP026520.1_prot_WP_042233824.1_492 [locus_tag=PC41400_RS02550] [protein=MraY family glycosyltransferase] [protein_id=WP_042233824.1] [location=556292..557404] [gbkey=CDS]
MYGLYVLGFVVSCLFALLLTPLVKRFAFWVGAIDKPNHRKVHTRIMARMGGVAIFLAFVGAYFVISPALSDVNGNAALAL
VLGGFVIVLTGALDDRFELSPRWKLLGQFVAACIVVACGLKINEFNVPFGGHFSLPDYVSMPLTVLWIVGVSNAVNLIDG
LDGLSAGVSGIGTGTILVLALLTPFSDAFTVILLCTILLGSISGFLFYNFHPAKIFMGDSGALFLGFALATLSILGFKVS
VVLSLLTPIMILGVPLSDTFFAIMRRFLNKKPISEADKNHLHHCLLQLGFSHRTTVLIIYGIAAMFGAVAVILNNYTDSA
WLTVLIVVILLFVLVVGAEAIGIISSSQKPVIRFLQKLFVKPSVNSRLPK
>lcl|NZ_CP026520.1_prot_WP_042233823.1_493 [locus_tag=PC41400_RS02555] [protein=S-layer homology domain-containing protein] [protein_id=WP_042233823.1] [location=557670..561539] [gbkey=CDS]
MKKHTRKWFSVLVLFSMLISLLPPLTAYAASTISITNMYNQTVNFGTDKEAPQDYNNVTRVTRNPFTLKASISNISDSQI
SSIYYEIMNVNTKQVTVEKSHKPTQTGANEITFSNITLSEGLNRIIVKMGDTNMLSSAPGWIYYTPTTNLTDFTVNNAGL
VDGKIYPVDRGNIKLGVTAVDIKAKASNANEVKAYETGSTVPKIGILNRTEGMVYFRGGDAAAGCGSTGEICLKAGDNRI
VFDSSNDSNSYQTEKRFVYDNGQPFAFGVEMGVTSAGTTQSLLDKPNFDVKELTLKTSLKVDYDSNGALRQNRVDVLING
QQVAAGLDLNTPSSSVGGAVKKFTTNSTFTRANEYAAYDLVLDLNVAGLTDWTAARNPQTIQFIFKDATGSAGQGESSYT
FNYTNPNEAGILDVSMVTGKDAATNTNIYTSLTESATTEINELPVEMKIKANGHTAKIKVFLDGKEIAEVGNPSNNEFIY
KLENITDGQKNIKFVPVDASGTLVENGSRQYTIQINNAPYIIANQIASPGNTGITLRDVKQLTCGSDVACLSGKAVNYLP
TGAGNKVELYVNDKMIELQASDFDANRQFKFDLARLGTKYGITPAQLFSEGANQIKFFMYVNNSLISKAIYNVNLITAEA
PEIVLFVPNEDSVGKSKYPKAQKPDMYATNERTVQLNGKFFNTTNAGNVTVSLNVYSKDKDGNPRADYHVWNNVSGSSTN
VSGGFFTGMPSSGVEFRTNAIKLPEKGEMTFQFTVTNQANMSVSKTVTIAREPLPYVVKKPVLIENEKHEYQANINSNFY
TIVIDAEKADSVTFGKETASKQTLNENGLDVDRFSYEVRDLKAGKNTIKFTVNRGTSKTNGTIVLYNTNTPVVGAEYLDN
MTSNFNVFNGDVKLKFDKDTKLMRFKPTFDNNQYITADRKVYFGIADTVDGRVDKINNPFSSDYAKSLLARKGKGDNFSA
ASKLFYIDSGTIPEKTNGQSTEDYLQNALRGRGMEPFDLKNNFYTRSVDQTVVPTKRGTLTLKFDQAIRDDAGKYISVFF
FDTYEQYNTLTGRGWVNLGGVVDVKSNTVTVPFEKFGYYQVMYMDKGFDDVTGHPWARNELETLFSKGIMNSKTNYSFSP
NEPISRGEFATMLVRIFDLPLNYSGTPTFIDVFKVENLYGLYDFKYIETAARAGIVRGTAGNRFSPDSSLTRQDASVMIA
RAADLKLGTDNKKILAGLQKTFTDANTIDYYAQPAVEAVVKAKLIEGKPNVLLEGQKKETVRFDPLETFTRAEAAIVAIR
VMKQQGKLK
>lcl|NZ_CP026520.1_prot_WP_042233821.1_494 [locus_tag=PC41400_RS02560] [protein=S-layer homology domain-containing protein] [protein_id=WP_042233821.1] [location=561970..564981] [gbkey=CDS]
MSNMSSRIPNHSQNPKDIQGGDKKVMKKSLSVILSTAMALSVFSSVAFAAKADDFSDLKDLDAATKAKFDSMIQAGIFDG
VEEGRFGLNDKMNRAQFAKVAALIFELKSSATTSSFEDVKSSDPANGYALPYIEALRVAGITDGKAEGKYVPAGDVTKEE
LATFLIRGLGKESEAKAKPGVSDKTVSDWAKGYVQLAQELKLFEAPAGQPFNGTAAATRDLLVTSAYQAQQVNNASKKLS
VSEAKATGVKTVTVKFNREVDTAKASLALKKGTASVATDVKWADDKKSAVLTLTNSKLSAGAYSVTLSGLAAETVDKAVA
EFSAEDEKLTKIEFLNPSDTIAYSKNVMIKLAATNQYGEAASLSAGAYTVYAGNNNDVLKRITKAEDGTLRLILDTATGD
NYKQGYGIVSVNIIENDSRVSVNKTFKIGTQPIITKSELGTVKYSTGDALSGKGQQATVELLRYDQYGNLVAADDSEDYD
KTITRFFSEYEKNLKTEVVKSGNDGILTVKVSLLDNVAKSGTYTLNVQTQASFSTVKINVKSSKVATKVKLGDVKKEISA
GDDEAIIDLAAFDSEGNQLSADDIVNSLKTNQGNDNTVNLSTSNGKIEIVQSGENKGKLRITDMTATDKSIISVSIVLTS
PGLNSFDTKTYNVKGARVASTLKLTNDPAKKALPGAKADIELNVLDQFDSKLDKAVTGDVYVQVSFTGSAGLTLVEDDQS
TAITTAKYTGEGLKTFNKGFDVKVAADVVNASNDLSGTFKAELVNTKGNVLASITRDVKVIDNKADLSYTVTPVSDLYAA
ADKLKDDQANIQKSALRKKVKITAKDNAGDLVALPDGVITSIATYNDSIVKASLQLDANGDPVPNAAPYVLGNKVGSATV
AVGYKTAKGEQKTASVSINVKDEAVQVAKVVGENDKADYTGAENAQTLLDLTVTDNYGVEYGKAVIADNNNVLGVIYTAT
DVQGTGSVTISSKGVITVNGSVKSFTLQAIANGKSDSAVITVK
>lcl|NZ_CP026520.1_prot_WP_042233819.1_495 [locus_tag=PC41400_RS02565] [protein=hypothetical protein] [protein_id=WP_042233819.1] [location=565050..565619] [gbkey=CDS]
MKKVIGLGMVILTVLASPIHAAAPTPGSADDPVVTKSYLEQYVQGLGGSVGTGGGSNGLTESKIKEIIAQEIAKLPTNNG
GGGTPSTGQGNAMKVVALKAGQTILADAGSELIVRNGKTLAVSNDDNGIPDVTAGKDLAAGTVVQNNHLLIFPRDGRGVK
PDLKETADVYVMVRGGYVVQNADGSKVPQ
>lcl|NZ_CP026520.1_prot_WP_009677087.1_496 [locus_tag=PC41400_RS02570] [protein=alpha/beta-type small acid-soluble spore protein] [protein_id=WP_009677087.1] [location=565849..566127] [gbkey=CDS]
MSKNNNKVVPESHKALDLLKYEIAAEMGLPVGKQALGSLDLSTEFASELGSIPATSIKEDYWGHISSRDSGAVGGAITSR
LIRKAEEVLFTL
>lcl|NZ_CP026520.1_prot_WP_042233817.1_497 [gene=metK] [locus_tag=PC41400_RS02575] [protein=methionine adenosyltransferase] [protein_id=WP_042233817.1] [location=566274..567482] [gbkey=CDS]
MTLALERGRRLFTSESVTEGHPDKICDQISDAVLDAFLENDPNARVACEVSVATGLVLVIGEITSRSEYVDIQAIARKTI
KDIGYTRAKYGFDYQTCAVLTSLNEQSPDIAQGVDQALEAREGQMSDAEIEAIGAGDQGLMFGYATNETPELMPLPISMA
HQLSRRLTEVRKNGTLEYLRPDGKTQVTVEYDGDKPVRVDAIVVSTQHSEEVTLEQIQKDIKEHVIKPIVPEEFLDENTN
YFINPTGRFVIGGPQGDAGLTGRKIIVDTYGGYARHGGGAFSGKDPTKVDRSGAYAARYVAKNIVAAGLAEKCEVQLAYA
IGVAKPVSISIDTFGTGKVSEEKLVELIRKHFDLRPAGIIKQLDLRRPIYKQTAAYGHFGRNDLDVPWERTDKAELLKAE
AL
>lcl|NZ_CP026520.1_prot_WP_042233815.1_498 [locus_tag=PC41400_RS02580] [protein=stalk domain-containing protein] [protein_id=WP_042233815.1] [location=567830..570448] [gbkey=CDS]
MRFTNKLWRSTLAFVVAFQVVVSLPVPTFAADPTVTLKSESILTSGAVMKKYVWNFTRSNKKVSATANVVEVDLTNPYVK
LDVIAGKNNQFTDKQTVATMAKAAGAVAAVNGDFFNTQAEGVPLGPQITNGQIMSTPSNKMSGLYAFGITKDNKPVIDLF
AFQGAVKAKDGTSFELGGINKTYYWYDDGTHSHTDGLFMYTDAWGQVDRSNDGKSVPTEVLVQDGVIKQIAPDTVIKIEP
PKNGYILRAAGKSAQFVKEHLKVGDPLTTDYAFINQRTGTAYANDAFKTMIGGHSILVDGAKATSFSRDVSSLGGYRSRT
GVGYSQDMKKAYLVTADKNDNSAGMSLQEFQRFLIQIGAYKAMNLDGGGSTQMVERPLGTNNIQLAHVTEYGTQRAVVNA
LGVFSTAPKGQPKGFTMKGDTELFLNEKATFTFSGYDEYYNPIVSDSVQPTWSVSNNLGKFDGNAFIPTSFGSGKITATT
GAGSSNLDVKVIRRADISSMKVSKASGQGLVAGGSYNLSVTATTKSGKTKEISPASLEWEVLGVKGEVKNGVLKVDSLEG
SKNAQVIARYDGYSSMLNIPLGNESMWYNLDDKSILTTSESYPAEVDTKLSIVKNESGNNSLQLAYDFTKGSGNKASYAV
FNNNGAQLYGYPQTINLKVKGDESQNWLRAEVIDADGKKELVELAKNINWQGWKSISANLSGLNLKYPLTLRSIYVVNPE
QGQDERALQGKIELDDISFSYPNYDTPSGSLNKVTLQIGNQMATVNGKSYWLEQAPINDRGNTLVPTRFVSEALGAKVLW
NQDALRATVVKDGNIVDMWNNELDLITNGKRVTAEVPPRIMNNLTMVPLRLLTETLGWKVTWNQAEQIVNLQ
>lcl|NZ_CP026520.1_prot_WP_174446847.1_499 [locus_tag=PC41400_RS02585] [protein=sensor histidine kinase] [protein_id=WP_174446847.1] [location=570537..571724] [gbkey=CDS]
MQVDALDRVIKNAIEVMESSKYQVFEICESARGERESLLRELGQVKDEVTQTIELVDNLERDYKRARIRLTEVSRDFARY
RENDIKTAYEAAISLQLQLTVSREKENHLKERRNDIQQRIKNVDKQLERAEMIVSQMNVVLEYLSGDLNQVTRILESAKN
RQLLGLKIILAQEDERKRMAREIHDGVAQTMANVVLRTEIAERMVTKQEYTAAKEELIDLKSQLRCGLEEVRKMIFNLRP
MALDDLGLIPTVRKFVQDFEDKTQVHTKLEISGRETRLPSGMEVAIFRLIQETFSNILKHAHATFVTLELSFQEDLVSIS
INDNGIGFNVDLTEHKIAKGNNFGIIGMRERLKLLEGDMAIYSEKDVGTKITMLIPIRAENKEEIGNGQPGESQA
>lcl|NZ_CP026520.1_prot_WP_009677083.1_500 [locus_tag=PC41400_RS02590] [protein=response regulator transcription factor] [protein_id=WP_009677083.1] [location=571696..572421] [gbkey=CDS]
MDNQGSPKRNVRLILADDHQLFREGLKRIINMEDDMEVIGECGDGIQVIELCNEMQPDVVLMDINMPLENGVVATERLKL
IFPDVKVIILSIHDDESYVFETLRKGASGYLLKDMEAEALINAIRSVVAGHAYIHPKVTGKLINQLRRMTYLDDNGVMGS
GNGVQESGLKYIHVPTSPLTKREAEVLRLMAEGKSNKLIGEFLYISEKTVKNHVSSILQKMEVDDRTQAVIIAIKNGWVT
L
>lcl|NZ_CP026520.1_prot_WP_042233813.1_501 [locus_tag=PC41400_RS02595] [protein=hypothetical protein] [protein_id=WP_042233813.1] [location=572472..572864] [gbkey=CDS]
MWTGLMLFVGSYIFCVCMLHAFKRPNEEVRSYHLLLVTKDNQLHIEWYIRSLFFLARFSGKSLSVSVIDEGSTDDTLRIA
EKLSRSYELKIRTVMPESSWEEWASSTERDNLMVMHVSNQKQLVKLPLFE
>lcl|NZ_CP026520.1_prot_WP_241688778.1_502 [locus_tag=PC41400_RS02600] [protein=helicase-related protein] [protein_id=WP_241688778.1] [location=574151..575188] [gbkey=CDS]
MPRSDGGSSGIAHSGKTAPSFLIWAVTGAGKTEMIFPLLQYELAHGRKIAIATPRRDVVLELQPRLMRAFEGRSIVTLYG
GGSQTWESGDITIATTHQLMRFAHAFDAVIIDEIDAFPYLNSAALEFAARKALAPGGRYVLLSATPPRALQREAASGRLP
HVTVPVRFHRHPLPVPKRVPAPPLRRILRSGALPQTLRARLEGSLERGAQLFVFVPEIKLVEPLVGLLQGLWPDKVIAGT
SSKDRERSVKVTDFRGKVTDILVTTTILERGVTVPKTDVFILDADSSLFNEAALVQMAGRAGRSKDDPRGLVFFAGLENT
VSQKGAIRQIKKMNRTARKRGYLLD
>lcl|NZ_CP026520.1_prot_WP_042233811.1_503 [locus_tag=PC41400_RS02605] [protein=ComF family protein] [protein_id=WP_042233811.1] [location=575215..576036] [gbkey=CDS]
MIGLPNPKPPAVWWSKVKRWFQSPEAETNARDRETANCVCCGGRRSSSKESLGLCMACFNAIPWIMAVRCPVCGRPDDCP
DCVRRTTCSFRYSRSAVRYDPLMKEWLASYKYRGNERLLPLFSGMLVHAYHQLVKELPAAGQEGSGFHCISYVPMNVSRL
AERGFNQAEAMAIELGRRLSLPVVPLLERTRHTAKQSFKSRGERLNDLQGAFKLLEEAIGSIVRLEEGITGQGWPVRVLL
LDDVYTTGSTLNQCAQTITQSLPVEVYGLMWAR
>lcl|NZ_CP026520.1_prot_WP_042233809.1_504 [locus_tag=PC41400_RS02610] [protein=flagellar protein] [protein_id=WP_042233809.1] [location=576164..576580] [gbkey=CDS]
MSMNVQNCPRCGKLFVKGFAETCPACVKDIEAQYVACADYLRQNRKCDLQELHEETEVPMRQIIRFIREGRISIADSPNL
TYPCEVCGNPIREHTMCDSCRQRLVRDVNQMMDEGKNNANAEKDAHHFRITDRLKDRH
>lcl|NZ_CP026520.1_prot_WP_042233807.1_505 [gene=flgM] [locus_tag=PC41400_RS02615] [protein=flagellar biosynthesis anti-sigma factor FlgM] [protein_id=WP_042233807.1] [location=576725..576988] [gbkey=CDS]
MKINDSHRIGNVNPYAKHKEQFQTAAAGKRNRPKDEVQISTEAKELLGAAGDENRMRRIEELKQQVKSGTYHVEAGKIAE
KLLPYLK
>lcl|NZ_CP026520.1_prot_WP_042233806.1_506 [locus_tag=PC41400_RS02620] [protein=flagellar protein FlgN] [protein_id=WP_042233806.1] [location=577014..577514] [gbkey=CDS]
MSVSEIVNVLNRIIECHQDLRELGTRKREAILQNSIEELSAIVQKESKAVKEIQELDQRRVVKINEYMVTRCYKTNPFIT
VGELVKLVFKAEEKQQLMDCQERLLKEIRQLKELNALNQQLVEQSLGFIDLSLDLLTATPEQDAIYANPTQQGQARRKNA
YFDTRA
>lcl|NZ_CP026520.1_prot_WP_042233804.1_507 [gene=flgK] [locus_tag=PC41400_RS02625] [protein=flagellar hook-associated protein FlgK] [protein_id=WP_042233804.1] [location=577566..579131] [gbkey=CDS]
MRSTFHGLELGKRGVIAQQVALNTTGHNITNANTPGYSRQVAQMVAARPLEYPSVTNSTSPGQLGMGVEVNAIQRIRDIY
FDVEYRNQNSLMNESKVISETLGNIEGVVNEPSDNGLAKNITAFWKAWQTLSKNPSTEDLAARSAVVQQAMAMADSFNSM
ANQFTTMGNNITERINIKVSEANSYVTEISDLTKQIKHIESGGDNANDLRDKRDYLVDQLSKLGNVKVTETSQTYQITFG
TKQVIDDLNSDTIVQADVAGVTSGEIFGYVQSRDTFIAEYTKQIDLMANTLATGKIEVTLPAGSILPNGMTPPAGAQMDP
TNPRKLAADAKVTVDGLNGLHRLGYTLEDPATNGKDFFVTSDGSGKFTAENIRVNPDIAKDLKKIAASLRIETVSKPPAA
PTNKVVQSNGDLALLIGGLAEKTFDFGGTALTASSTYGGYYEAVMSQLGVQTDHAKKMLENGELLTDHADSLRQSISGVS
PDEETANLIKFQHAYNASARVITVVDEMLDKLINGTGVVGR
>lcl|NZ_CP026520.1_prot_WP_042233802.1_508 [gene=flgL] [locus_tag=PC41400_RS02630] [protein=flagellar hook-associated protein FlgL] [protein_id=WP_042233802.1] [location=579145..580056] [gbkey=CDS]
MRITQNMMSSSMMKNLQTNYNRLDKYQEQLLTQQRINRPSDDPVGISSSLRYRGEIDMTDQFAENSKDADSWMNFTDTVM
NEAVKILQTVSEKAVQGATESNNPEARKILGQEIDELYKQMVAIGNSQFKGKYIFNGQTTDQKPFSDDPANTAYSLDEGI
MRYQIGAGIYVDVNSLGKNVFGPFQDPADPTKAANVFKVLDNLKTALTGDNTAGIEKSIKEIQTTMDQVLFAQADVGARQ
NRLQFTTDRLEDLTLNYTDLQSKIEDVNVPKTITDLKTAESVFQASLDTMGRIIRPSLLDFLR
>lcl|NZ_CP026520.1_prot_WP_042233799.1_509 [locus_tag=PC41400_RS02635] [protein=DUF6470 family protein] [protein_id=WP_042233799.1] [location=580069..580641] [gbkey=CDS]
MQIPRIQISQQYTKIGLSTTEGKLSIEQPKAELNLQQDDTKVSMETEKSSLEIDSTRAWSALGSSKLIEMRDLIAQSAFD
ISMQNIAEIAQNGDRMMEITNHSNAFADIARQNAFKDRPVQVTAEPSYDNVDIQYAPSVIHTTVQDGKITFDPELNSPRI
QYEAGKLEAYVTQKNAISFSVVGGNWDASI
>lcl|NZ_CP026520.1_prot_WP_042233797.1_510 [gene=fliW] [locus_tag=PC41400_RS02640] [protein=flagellar assembly protein FliW] [protein_id=WP_042233797.1] [location=580705..581136] [gbkey=CDS]
MIVQSTRFGEIDVSDEQLITFEAGIFGFETNRRYALLQLDNEDPTFHVLQSLSDADLAFVIAEPFLFEPAYEFELPKAIS
EQLEIKNEGDTQVFGIVSVRAKDNVTMNLKAPIILNKKRNKAAQIILESTNHPIRHSLIKGGA
>lcl|NZ_CP026520.1_prot_WP_042233794.1_511 [gene=csrA] [locus_tag=PC41400_RS02645] [protein=carbon storage regulator CsrA] [protein_id=WP_042233794.1] [location=581138..581365] [gbkey=CDS]
MLILSRLKGEKIMIGQDIVLTVLDVAGDQVRIGIEAPKDIPIHREEIYKAIQTENQAAVKMDLNAASLLKAVKGK
>lcl|NZ_CP026520.1_prot_WP_042233793.1_512 [locus_tag=PC41400_RS02650] [protein=flagellin] [protein_id=WP_042233793.1] [location=581531..582370] [gbkey=CDS]
MRINHNISSYNAQRQLGSNNTAQSKSLEKLSSGFRINRAADDAAGLAISEKMRNQIRGLEQASKNSLDGISLIQTAEGAL
NETHAMLQRMGELFTQAANEVLTTSDTAKIKTEVDALSNQIDAIANQTQFNTKKLLDGSTTTLKFQVGANKGETIDLTLV
SMKAATLSVDAGSLKDISASAAADINKDAAENMELVQKAIDTVSTTRSNLGAVQNRLEHTINNLGTTAENLQAAESRIRD
VDMAKEMSEFTKNGILQQAATAMLAQANQQPQGVLQLLR
>lcl|NZ_CP026520.1_prot_WP_053228778.1_513 [locus_tag=PC41400_RS02655] [protein=flagellar protein FlaG] [protein_id=WP_053228778.1] [location=582444..582782] [gbkey=CDS]
MKINPGTASTSSTPLFNTISATLLSDATNENLTPSVSADTQNIQQLVDRVNKALHETDTHIKVKVHEKTNTIMVVVVNND
TDEVIREIPSERMLDLMYNMCVNVGVFLDKKI
>lcl|NZ_CP026520.1_prot_WP_042233791.1_514 [gene=fliD] [locus_tag=PC41400_RS02660] [protein=flagellar filament capping protein FliD] [protein_id=WP_042233791.1] [location=582797..584302] [gbkey=CDS]
MAGVSGPTRLTGFSGQLDTDSLIKKLMDAEKVPLNNLLKKKQMTLWQREDYRTMNTSLLSLRKTIDDLRFENKFEAVKSN
SSNPTVLEVTSSGTNPGISNVKITQLANSGSIISDKVTKLPTEQLNVSGSFTIEGSDPNNSVQITVSADSTIDSLVKQIN
GSAAKTGVKASFDRSSGVLYLNSTSTGAQSAVKVTDDANGTFKNLFGNIQLTSQGQDAKYNVNGKDITAKTNDVIINGVQ
VSLRSQGEATIGAVADHSAVADKIKNFVKQYNELIDQYTDATTAKKNRDYKPLTAEEKEGMTEAQIALWEKKARQGDLYN
DGMLKETLQTLRSALNIPLEGITDPADPKSQLRLLSQIGITVKNDFRENGKLEVDEAKLNDALNTRFKEVVQLFTKTSST
PSDTPENIKKRRSEMGIGDRLYEVMTAQLAKFTKKIGLGSNESLDDSVLGKQLKEMGTKETDLQRKLREIEDRYYKKFSA
MEKALQKLNSQGSWIGSQLKG
>lcl|NZ_CP026520.1_prot_WP_042233789.1_515 [gene=fliS] [locus_tag=PC41400_RS02665] [protein=flagellar export chaperone FliS] [protein_id=WP_042233789.1] [location=584348..584740] [gbkey=CDS]
MQNAKQQQYLKVRVETASPGELTLLLYQEMVKLLLQAKQLYSQNQYEEMNQPLYKVKSILSELIVTLDMKYDISRDLQNL
YLFYDQHIANFIIKKDESILNDILDFARGMAETWKQALASLKTGGYQENA
>lcl|NZ_CP026520.1_prot_WP_042233788.1_516 [locus_tag=PC41400_RS02670] [protein=hypothetical protein] [protein_id=WP_042233788.1] [location=584733..585062] [gbkey=CDS]
MRSSSLLLEELAGLYKQVLEAPSFDRYEQVSEKMDQLYLDLSEKSFTAADKLTLEQIQNMHEKVIAVIQQEQKEIKNQIN
TMEMKKNVSNAYSTKASYTNDAFFVDIRN
>lcl|NZ_CP026520.1_prot_WP_084706671.1_517 [locus_tag=PC41400_RS02675] [protein=IPT/TIG domain-containing protein] [protein_id=WP_084706671.1] [location=585081..589049] [gbkey=CDS]
MKKTVVKSIRLILSLILLLSVFPVTFASAEAADEIVRVTKQVNPTSILEGGEADVTVNVQGSDGVNFVKPNDIILIIDRS
GSMLPQNNNGEDKMTNAKNAAKGFVDLIDFSKHRVGVVDFASDVKSKDLSSNPADIKNYITGINASGGTATMSAIKKAQE
LLRAHRTDAQPVILLLTDGEATEPAPVDNARKVALEQANAAKGEGTVFYTIALLKKTENPDTSAPNLLMKDMATTSQHHH
FVLGSVGLAEIYQAIVSEIGVASAYDVTVKDSVAPEFEIVPGSYKDNIPQPTVSGNTLTWKFNELKKEKLTFNYKIRHKA
GQRTGDLSVGSQDIDVTFKDYLGVPSATKVVNPLIKVSYPAPEITSIMPDKGSVIGGETIVLNGKNFRTNPAVVFGNKDA
VSVQFVDSTKLVVVAPPGNQGAVELKVTNNDGQSATVPYYYFAVPEFISVTPSQGVLAGGNDVVITGKYFLANPKVKFGQ
HEAVIKSSTSTQITATVPAGDVEGFVDIEIVNSDNTSVTAKNAYEYLVGPSVTEVTPNQGSSKGGDLVTIKGSHFKSGAV
VKFNTTAVPTEFVSATELKVTTPAWPRAESVKVTVINPDKLEGYLDQAFKYLNPAPTIESVSPNSGLVSGGDLVTITGTD
FLPDAKILFKDKEMASSFVDSKHLRVRSPKWTQEETVDITVTNPDGQSAVLNQSFSYKFPQAAELVSLSPISGPLDGGNV
ITVKGTNLSSDLSLYVADTKVLYTLLSSDTLTFKAPKTAVAGKVDIKVTDTYNRESILKEAYEYLAPAPLPAPTISGVTP
SQTVRTGGEFIYVNGDNFQEGAVVWLNDSKQTTSFMSTKQLRFRAPVWGKEEIVTVKVVNPDSQFVVSDGALKFTNPPAD
PAPEISSLTPDSGEIKGDYFVTINGKYFKSGAKVTFGGKELTTSFLSDTQLRVRAPQGEKEGPVEVKVVNPDGLQAISTF
TYTAPVLGPAPVITTLSPAQAPVTGGIFVVINGENFNAQSSVKFGDKVITTSFLSDKQLRVRIPVWTKAEKVDVSVSNAD
GQVAVSVGGFEYIPVAPAVVTGLTPNHIQMDLGGFVVIDGANFDPATKVYFGDKEASVSFLSDKQLRARAPIWSKAETVS
VKVVNPDGQITTVANGFTFDPIPLKPAPTISTLSPNHSLQTGGDLVYVNGSNFIEGAKVSIDGGTPIVTSFFSSNQLRFR
IPASSKTGPVDVKVINPDGQFVILANGFTYDAPPMKPAPTISNLTPNNALNSSGELVYINGSNFMDGAKVSVDGATPIAT
SFLSSTQLRFRIPITSKTGPVDIKVINPDGQTVTLVGGFTYR
>lcl|NZ_CP026520.1_prot_WP_042233786.1_518 [locus_tag=PC41400_RS02680] [protein=glycosyltransferase family 2 protein] [protein_id=WP_042233786.1] [location=589113..590201] [gbkey=CDS]
MKQPISACIMTKNEQDTIGNCVDSIRPYVEEIIVLDTGSEDATADIAFSKGAAVFHTVWQNDFSKVRNELIQLAKQPIVL
IIDADEIAVPKQERAFSEIGKRLLSDESLVAKISILNETDDGISSASISRIFNKDSGIFFKGVVHEQIECSTKELKFIQS
DIVLRHTGYLSSQIIANNKYERNLTLLFKELEKNNKNSYIHFQIGRTYSASGAYEKAAHHLNLAYKFSSDEYAYHSALIQ
AYGWALLRTKKFSELFELLQIGLQLYPDYTDLYFLYGCTLVELKSAEHVHLIPDIFLGCIELGEPDPQKYETVSGVGSFK
AYYNLGLYYELTGQTDRAFIAYKNSAEADFKPAIERLTQLRT
>lcl|NZ_CP026520.1_prot_WP_042233783.1_519 [gene=fliB] [locus_tag=PC41400_RS02685] [protein=flagellin lysine-N-methylase] [protein_id=WP_042233783.1] [location=590219..591457] [gbkey=CDS]
MKQNTLIPEYMKKFQCIGGDCEDTCCAGWTVSIDKQTYKKYQKATDPELKTKLNSFIKRNKNTATKSDRYYASIIMGKDK
HCPMLTEEKWCSVQLKMGEGALSTTCRMYPRNLNKYNDTFEFSGRVSCPEIARLVLLEPKGIEFDETEYEIQPNWPVIKH
HGNDGAADYTQHFWPIRMLAIEMIQTRTIALGDRLVLLGLFIDALQKDIDQGLGLGAADIIESFRSKLMSAEYLESISKL
SVNLDLQLQALFELIKSRLTYGATMDRYHVTFTEMLEGLKLAEADFKLDEFRVNYEENFNAYYSPYMKEKEYVLENYLVN
FLFNSLFPNLSVKNNSLFDEYALLAVMYGMLKIHLVGVAGKHKVLNDEIVVRTVQSFAKAIEHTSVYLQSIVEELKRNNY
YTLAHISLLVKD
>lcl|NZ_CP026520.1_prot_WP_042233781.1_520 [locus_tag=PC41400_RS02690] [protein=glycosyltransferase] [protein_id=WP_042233781.1] [location=591495..593252] [gbkey=CDS]
MLQKLRSHLQYKRMEDVRAETCPKRPLQAKKMSPKHRQHIVFVMNQVGIWGGVKIVFEQANRLIEKGFRITILSRHPFPE
WFVLQAEYIQIPYGLDLGQEIPLCDIIVATYYDHIYECVQTGIAPVVYFEQGDKHLFEPEKLEEEQLQFIQKQLERADRI
LTVSHQVSNLLKDLYNCDSTVIHNGIDSDIFNRRDYAPERIRPYILMMGNEQIAFKGIQDVWKAYLHSPATQENYDLIWI
TPVKPQFLPQEKPWISEIYVNPTQQEIAELYRGAAVFVSGSHYESFSLPVIESMACGCPVVSTRNQGVLEYSAEGSNILL
AEIGDPESLSKQLTWLLENPLLRESQIVKGLETAARFSWEAVLPQVEMYYKEAAGYDPVPLNTLDEWNILLQKKDFMTEE
EHERFIKRLLHSSKDQIELVSVYSLIENHSFGRWETAAVRKQPGSRVSVDKVYSLARNTISPESDHDAAIGLLAQGKPDE
AYHLWQRLINTNDSSPEQQGIYTKWLILCLIELERDEDALQLLNRALPKYPDYTDLYYLYALVLTLLGKGSEAAPVKGIA
KLVGDAAHYPEFFLNSSTLLKNLQQ
>lcl|NZ_CP026520.1_prot_WP_042233779.1_521 [locus_tag=PC41400_RS02695] [protein=FkbM family methyltransferase] [protein_id=WP_042233779.1] [location=593277..594065] [gbkey=CDS]
MHGTYIGNNKMMVKLAYNAVVTVPSNDLSLMPTLVTTGMFEVPLTKYFLTQLKPGYTVVDVGANIGYFTVLAAKVVGNTG
RVIGYEADPNTAAVLKDNLAMNWLTENVTIINKAVYSENKTLQFRSSERFHGYSSWKEKPQDDNLIDHYTIRDVAAVSLD
NELQELESIDLVKIDIEGGEYHAFLGMKKLIQDRKIKTITFEWNKAMLGTDEGPFLDLISDLITNRGARLTLLDKEGEVV
ETTLKDLAAFEFYPFVLLEFQN
>lcl|NZ_CP026520.1_prot_WP_009677088.1_522 [locus_tag=PC41400_RS02700] [protein=cold shock domain-containing protein] [protein_id=WP_009677088.1] [location=594310..594507] [gbkey=CDS]
MQGKVKWFNAEKGYGFIEREDGGDVFVHFSAIQTDGFKTLEEGQAVEFDIVEGARGPQAANVNKL
>lcl|NZ_CP026520.1_prot_WP_042233776.1_523 [gene=raiA] [locus_tag=PC41400_RS02705] [protein=ribosome-associated translation inhibitor RaiA] [protein_id=WP_042233776.1] [location=594705..595253] [gbkey=CDS]
MKFTVRGDHLEITAALREYVEKKLSRLEKYFEAPLNSEGYVTLSVVKNIHTVEVTIPLQGVLLRAENKDQDMYAAIDLVV
DKLERQIRKHKTKMNRKIRLEGNKRELSKLESPVLTMDEEEEDYELVRTKRFTLKPMDIDEAILQMNMVGHDFFVFSNMD
TEQVNVVYKRDDGRYGLIEPAI
>lcl|NZ_CP026520.1_prot_WP_042233774.1_524 [gene=secA] [locus_tag=PC41400_RS02710] [protein=preprotein translocase subunit SecA] [protein_id=WP_042233774.1] [location=595750..598254] [gbkey=CDS]
MLGLVKKMFGDSNEREIKRYWKAVDAINGLEPEISALSDEQLKGKTVEFRERLEKGEDLDDLLPEAFAVVREASKRVLGK
RHYDVQLIGGMVLHEGKIAEMRTGEGKTLVGTLPVYLNALAGKGVHVVTVNDYLAQRDSGEMGQIYDFLGMTVGVNLHDL
SHEQKQAAYTCDVTYGTNNEYGFDYLRDNMVLYKEQMVQRPLYFAVIDEVDSILVDEARTPLIISGQAAKSTDLYYAADR
FVSKLEAEKDYTVDIKVRSVSLTEQGVEKAEKAFGIENLFDHQNVTLNHHVTQALKARVIMRRDVDYVVQDDEVLIVDEF
TGRLMTGRRYSEGLHQAIEAKEQLQVQNESMTLATITFQNYFRMYRKLAGMTGTAKTEEEEFKKIYGLDVIIVPTNRPMI
RKDMPDVVYKTEMGKFKAVVEEIVERYKNKQPVLVGTVSIENSELLSEMLKKKGVQHKVLNAKYHAEEAEIVSRAGQAGA
VTIATNMAGRGTDIMLGDQVHEIGGLHIIGTERHESRRIDNQLRGRAGRQGDPGSSQFYLSLGDELMKRFGAENIMAMMD
RLGMEEDQPIESKLITRAIESAQKRVEGNNFDMRKVVLQYDDVMNQQREIIYKQRREVLESENIREIVETMLRSSIERIV
NAHCSESQIPEEWDLQAVVDYVNNNFLSEDALNSKDIWGKEIDEIIEFIDEKVKDKYNEREEQIGAEFMREFEKVVVLRA
VDSKWMDHIDAMDQLRQGIHLRAYGGTDPLREYQFEGFEMFHEMVDSIQEEVATYIMKAHVESNLERQAVAEGQAVDPKV
ESAGKQPIVKEEQIGRNDPCPCGSGKKYKQCHGA
>lcl|NZ_CP026520.1_prot_WP_129112445.1_525 [gene=prfB] [locus_tag=PC41400_RS02715] [protein=peptide chain release factor 2] [exception=ribosomal slippage] [protein_id=WP_129112445.1] [location=join(598452..598523,598525..599565)] [gbkey=CDS]
MLEPEVKQDLREIAKRLTELRGSLDLDHKLEEIGDYEVKMAAPDFWDDNEKAQKIIGELNAIKSVVDQFQALNNEYEDLD
VMAQLADEENDEDMAAELADGLKGVLRKLEDFQLQLLLNQPYDRLNAILELHPGAGGTESQDWAEMLLRMYRRWAEKRDF
KVEVLDYLPGDEAGVKSVTLLIKGHNAYGYLKAEKGVHRLVRISPFDASGRRHTSFVSCDVVPEIEDDVEVDIRTEDLKI
DTYRASGAGGQHINTTDSAVRITHLPTGVVVTCQTERSQIKNRERAMKHLRSKLYEKKIEEQEKHLAEIRGEQSDIAWGS
QIRSYVFHPYSMVKDHRTTEETGNVGAVMDGDLDPFIDAYLRLQIQKKEE
>lcl|NZ_CP026520.1_prot_WP_042233772.1_526 [locus_tag=PC41400_RS02720] [protein=YitT family protein] [protein_id=WP_042233772.1] [location=599915..600799] [gbkey=CDS]
MAMAKGKRRKARLVLNSPQQKVYEYAMLLIGCLIISISFNVFLNPNQVASGGVSGLSTIIQQTLGWQPALTQWGLNIPLF
LLGLWLLDRSYSLKAVVGSIVLPLCVMLTSHLPTLTHNILLATIYGGMVLGLGIGIVFRGRGSTGGFSISSQILHKYSGL
SLGVSVAVFDGMVILLAGIFFGPEKALYALIGLFVTTKTIDIVQVGLNTSKVAYIISNDPEKLREAILHDLDRGLTQLNG
AGGYTGDARLVLMVVVSQSEVSKLKALVQSIDPQAFIILSNANEVLGEGFKRAV
>lcl|NZ_CP026520.1_prot_WP_042233770.1_527 [gene=argC] [locus_tag=PC41400_RS02725] [protein=N-acetyl-gamma-glutamyl-phosphate reductase] [protein_id=WP_042233770.1] [location=601049..602098] [gbkey=CDS]
MTEHVRAAIVGSTGYGGVELIRVLLGHPAVTITSVISSSSAGAPISEGFPHLNSIMHDSLDAVDVPLIASKADVVFLATP
AGVAAKLAPQMLDAGLKVIDLSGDFRLKTPEIYTQWYKREGASQEDLDKAVFGLCEIYGEQVPGKEFISNPGCYPTATLL
GLVPAVKAGWIDPSTIIVDAKSGISGGGRGLGLGFHYAEANENFSAYKLNRHQHIPEIEQVLSDAAGREVIITFSTHLVP
MTRGIMATCYAQLNGKHDTREFIEMYRNFYEGRRFVRIREEGKWPATKEVWGSNYCDIGFAVDERTNRVTVVSVIDNVMK
GAAGQAVQNLNLMLGWDETTGLQLTPVYP
>lcl|NZ_CP026520.1_prot_WP_042233768.1_528 [gene=argJ] [locus_tag=PC41400_RS02730] [protein=bifunctional glutamate N-acetyltransferase/amino-acid acetyltransferase ArgJ] [protein_id=WP_042233768.1] [location=602252..603478] [gbkey=CDS]
MTLLGRFTVVPDGTVTTPQGFRAGGLHCGLKKTERNDLGAIVCDVPAAVAAVYTLNAFQAAPLRVTQESIAAGGRLQAML
VNSGNANACTGEQGLADARAMRDALARRLGIAEELVGVTSTGVIGELLPMTKVLAGIEALPEKVNAAGGEDFSQAILTTD
LVKKVICVRVNIGGREVHIAGAAKGSGMIHPNMATMLGFVTTDANIESKHLQQLLGEATDSTFNMITVDGDTSTNDMVVV
MASGLAGGEALQPGHADWEDFAAAFRHVSEHLAKAIARDGEGATKLIEVNVHGAVSQEAARAIAKTVVGSSLVKTACFGA
DANWGRIIAAVGRAGQPVNTETVDVSLGGITVLKQSKPVAFDEDRAEAYLKGDIVQIDVHLNMSDSGATAWGCDLTYDYV
RINAAYRT
>lcl|NZ_CP026520.1_prot_WP_042233765.1_529 [gene=argB] [locus_tag=PC41400_RS02735] [protein=acetylglutamate kinase] [protein_id=WP_042233765.1] [location=603524..604309] [gbkey=CDS]
MESFVMKCGGSTLAALPDSFFTDLKQLQDRGIVPVIVHGGGPAISETLNQLSIASEFVGGLRKTSEAVLDVVEMVLAGRI
NKEIVRRIQLSGGKALGLSGVDGRLITARPVENTSEVGLVGDVTDVNAELIHGVVGMGYMPVIAPIGLGGDGNQRYNINA
DTAAGAVASRIGTDRLIVVTDVPGIMKMEDGVKRVLTTVTIADIEAMIESGDIYGGMIPKVRAAIQCIQGDVREVVIVDG
SVPGILSSVLDGEQVGTRIIR
>lcl|NZ_CP026520.1_prot_WP_042233761.1_530 [locus_tag=PC41400_RS02740] [protein=acetylornithine transaminase] [protein_id=WP_042233761.1] [location=604351..605541] [gbkey=CDS]
MVQQASSLFPTYARFPITLVKGEGSRLWDDQGKEYIDLMAGLAVTNLGHAPEVVKEKLVAQLDQLWHVSNLFHIPNQEKL
AKLLTDNSCADAVFFCSTGAEANEAAIKLARRYSQKVLGENRYEIITFHQSFHGRTLATLTATGQEKVQEGFHPLPEGFV
YADYNDLASVEKLISERTCAVMLEMVQAEGGVIPADPDFVKALAELCKERGLLLIVDEIQTGMGRTGKLFAYEHYGIEPD
IFTLAKGLGSGFPIGAMLGNAKVAPAFSAGSHGSTFGGTPIATAAAIATVEYMLNEKIPQRAAELGEWTVKTLREKLADN
PLVEAVRGEGLLIGIVCKQPAAELIAEIHEAGVLVVSAGPQVVRLLPSLVIPKEDLQKGLDVVCDVLARKASALVQ
>lcl|NZ_CP026520.1_prot_WP_042233756.1_531 [gene=argF] [locus_tag=PC41400_RS02745] [protein=ornithine carbamoyltransferase] [protein_id=WP_042233756.1] [location=605608..606564] [gbkey=CDS]
MSSTVQEELQQQLKGRDFLGLVDYTPAELNHLITLARDLKAKQKAGEEHQLLKGKTLGMIFEKSSTRTRVSFEVGMFQLG
GQALFLSKNDIQLGRGETVSDTAQTLSRYLDGIMIRTFAHRNVVDLARAATIPVINGLTDFSHPCQALTDYMTIQEHKGR
LEGLKIAYVGDGNNMVHSLLVGAAKLGLNITVASPEGYTPDEELVRSSREFASQSGGSVLVTQDPKEAVADADVIYTDVW
ASMGFEAEQQERERAFSGYQVNEELARYAKKDYLFLHCLPAHRGEEVSEGVIDGPNSFVFDQAENRLHAQKAVMASIM
>lcl|NZ_CP026520.1_prot_WP_042233754.1_532 [locus_tag=PC41400_RS02750] [protein=argininosuccinate synthase] [protein_id=WP_042233754.1] [location=606605..607843] [gbkey=CDS]
MAKEKIVLAYSGGLDTSVILAWLKETYDAEIIAFTADIGQKDELDGLEEKALATGASKVYIDDLREEFASDFIYPMFQSG
ALYEGQYLLGTSIARPLIAKRMVEIARAEGATAIAHGATGKGNDQVRFELTAAALAPEISVIAPWRLEAFREQFPGRAEM
IEYAEKHGIQVQATASKPYSTDRNLLHISFESGMLEDPWFDASADSNRDMYVLSVSPEDAPDQPEYIELEFEQGNAVAIN
GEKLNPLQMMETLNELGGKHGIGRIDMVENRFVGMKSRGVYETPGGTILFAAHRRIESLTMDREVMHLRDSLISKYSTLV
YNGFWFAPERLAIQALVTESQKNVTGTVRMKLYKGNVIAAGVKSPVSLYNPNIATMEADPTQAYDQGDAAGFIRLNALRL
KVSSGVEQQSGK
>lcl|NZ_CP026520.1_prot_WP_042233752.1_533 [gene=argH] [locus_tag=PC41400_RS02755] [protein=argininosuccinate lyase] [protein_id=WP_042233752.1] [location=607944..609359] [gbkey=CDS]
MSKLWGGRFTKQTDQLVEEYTASITFDKELAEEDIQGSLAHVTMLGKCGILPQEDVETIKDGLNRVLGMIRRGELEYSIA
NEDIHMNIEKTLIDEIGPVGGKLHTGRSRNDQVATDMHLYLRGRVVEFVSLLQKLQEALLTQAKANLDTIVPGYTHLQRA
QPILFAHHLMAYVSMFQRDIERLQDSYKRINVLPLGAGALAGTTFPIDRHFVAEQLGFEGVYENSLDAVSDRDFILEFLA
NASFIMMHLSRFCEELVMWSSTEFNFVELDDAFCTGSSIMPQKKNPDVAELVRGKTGRVYGNLFGLMTVLKSLPLAYNKD
MQEDKEGMLDTVRTLQGALQLFAPMISTMKVNKDRMRQAVNQDFSNATDIADYLVNKGLPFRQAHEVIGKTVLYCIQEGK
FLLDLSMEEFNKFSELFGDDIYEVLQPEQVVNARNVYGGTASSQVEGAIARAEKALEVTVSWVDSKLDKSK
>lcl|NZ_CP026520.1_prot_WP_042233750.1_534 [locus_tag=PC41400_RS02760] [protein=VanW family protein] [protein_id=WP_042233750.1] [location=complement(609488..610930)] [gbkey=CDS]
MKPVKRVLTTIAIVLPLAAVVGLGSIAFYGSRPTFPEQFSVSGWKVGGMAYADFPRQLSRQVALLNSVPVQLLSNRKDVP
SRSQKLQDLGLRVDDRELQALLHDLTKGNLLDRARNRWKLRHYEAKIGLSIEADQLDKKLQSLWPELYAVQPVSAQRIVT
PDDRVQYKAGQPALRVDTSRLKADIERQIRQIDPKRAENPLSIQVPFRLQEPEVTEKMLRGQHVDRLITQFSTPLNQSSP
GRRHNVRSTAATLHEVILKPGEIFDYAPIIEQTGVKFGFREAPVILNGKLVPGVGGGICQVSTTLYNAVLRAGLEIVERR
NHSLPISYAPLGQDATFSTGFINFRFRNDTKAHLLIRTEMSDTALTIKLFGSVPTDTTYEIMSNTVETIQPPVKYVHNPG
LKTGEQVELSPGKQGFVVDTYRIKKQGGKEVSREKISRDRYTPQARVVASNTGVLKPPVESAPPQKNVIEDGVTQNGSTP
>lcl|NZ_CP026520.1_prot_WP_042233748.1_535 [gene=ftsE] [locus_tag=PC41400_RS02765] [protein=cell division ATP-binding protein FtsE] [protein_id=WP_042233748.1] [location=611190..611876] [gbkey=CDS]
MIEMQDVWKTYSNGTHALRGIDVKVDRNEFVYIVGPSGAGKSTFMKMIYREERPTKGQIFVNGFNIEKLKQRKIPYVRRN
IGVIFQDYRLLPKMTVYENISFAMEVIEAPKKLIKKRTMEVLDLVHLKDKANSLPGQLSGGEQQRIAIARAIVNNPAVIV
ADEPTGNLDPETSWGIMKLLEEINFRGTTIMMATHNKEIVNTMRKRVIAIENGIIARDELRGEYGYEE
>lcl|NZ_CP026520.1_prot_WP_042233746.1_536 [gene=ftsX] [locus_tag=PC41400_RS02770] [protein=permease-like cell division protein FtsX] [protein_id=WP_042233746.1] [location=611866..612783] [gbkey=CDS]
MKSRTMARHLREGFRNIIRNGWMTFASISSITISLFILGLFLLLSLNVNYVADQIEKQVEIRVYMDLNITEDQTNQLKQE
LNALPEVQKVTFVSKEQGLEDLRNRLGEEGKGFLEGFEGDENPLPPAFKVEVKDPRQVAGVADTINKLNEGKEPKPIGSV
KYGQGTVENMFKITEIVRNVGLVLVALLAFTSIFLIANTIKLTIVARRREISIMKLVGATNAFIRWPFFIEGALLGIGGS
AIPVAVLLYGYNKLVESSKSELSMVFIKLKPFAEVSYVIAGLLLCLGFLIGIWGSIISVRKFLKV
>lcl|NZ_CP026520.1_prot_WP_053228777.1_537 [locus_tag=PC41400_RS02775] [protein=M23 family metallopeptidase] [protein_id=WP_053228777.1] [location=612828..614093] [gbkey=CDS]
MKKRLMPLVVALGISTTLLPYAVANGASNSAKIDQEISNMKKMKSAAEQKAKDVQNQLTQVQQEKQETAKDIETLMNQVD
SATAKLSGLNVQIENTTEDLEDNAKQLDEAEKRVETRDELLKKRMRIMYENGSVSYLEVVLNSTSFSDFIDRMSALKSIV
GQDKEILEANKKDRALVAEKKQQIEKQLAYVKNLYAQTETLKQDLLVKEKAKEVKIASLSAEQQHLEEISSDQEQAIMEY
AKKESQLQAKKAADAKAAAAAAAAAASRNSSSIGGSSGSSGGGPATGSGGSMGYPLPKRVALSSGFGGRVDPINGSFAGH
TGIDMPSPQGTDIYAAESGTVLVASEWSGYGNCVIIDHGGGVWTLYGHIRNGGIKVSKGQSVSRGEKIAEVGSTGRSTGN
HLHFEVRINSKPVNPSSYIGY
>lcl|NZ_CP026520.1_prot_WP_042233744.1_538 [locus_tag=PC41400_RS02780] [protein=S41 family peptidase] [protein_id=WP_042233744.1] [location=614189..615643] [gbkey=CDS]
MQFKGRTVIAFVLLGMFASSILTLTFTGRNLLPLGSKSVGASATAGSSGLSSQDVGKIAKTYQLIENKYLNKVDHDKIMN
GAINGMLESLEDPYTVYMDSNEAKQFSESINSSFEGIGAEVTSEDGKIKVVSPIKGSPAEKAGIMANDVIVSVNGEKLDG
LTVNQAVVKIRGQKGTQAKLEILRQGSTEPVQVIVVRDEIDIETVFGEMLEGGIGKIEIRQFSANTAEHFQEELKKLETQ
GMKGLVIDVRNDPGGLLNVVIDIVEPFVPKGKPIVQIENREGKREATTSKDGSKKYPVVVLTNKGSASASEILAGAFKEA
VGSKTVGETTFGKGTVQETYVKEMGDGSNIKMTVYKWLTPDGNWIHKKGIEPDIAVEQPAFFKASILSKKDTLKPDSNSD
DVKNLQLMLTGVGYEPGRTDGYYSSKTADAVKAFQKSSSLPVTGNVDTATAKKLESAVLEAVKLPKNDNQLKAAVKELQT
QLSK
>lcl|NZ_CP026520.1_prot_WP_042233743.1_539 [locus_tag=PC41400_RS02785] [protein=PDZ domain-containing protein] [protein_id=WP_042233743.1] [location=615819..617108] [gbkey=CDS]
MDIIAEIAGRFLTAFGRLWLNPFTYIGLLLVLLQYRKQIQFERKLFATRLHSLIGETWRTYLWGWVAGLGVSVVMLFVGV
SLQADTILLLWGLSVAMILIRIRFLCLAYSVGLLGILQVVLAGLPGPSGDGFWSSLAGVVKNADVPSLLALVAVLHAAEA
LLIRSQGARMASPMFYEGKRGKIVGGYQLQGFWPVPLFLAVPVQAGTGAVDFPWSPLFTGDALTAGWSLLAFPAMIGFTE
LTMTRLPKEKVGRSSGMLLAYAAVLFLSALLAQVWSPFTVVAAVLGIVLHEAVIWYGRWEEAQRSPCFINQPRGLKILAV
VPGSPAAELGLRPGELVHKVNGVRVHSKTELHQAIQVSSAYCKLEIVDMEGEIRFHKRAVFAGEHHQLGIVICPDQDAVY
YVEIRQLPLLTYFTRKLTGLLTKRSEETL
>lcl|NZ_CP026520.1_prot_WP_129112446.1_540 [locus_tag=PC41400_RS02790] [protein=flagellar motor protein MotB] [protein_id=WP_129112446.1] [location=complement(617268..618095)] [gbkey=CDS]
MRKRGKKQEEHVNHERWLITYSDLITLLLVFFIVMYAMSKVDINKYETMAKALNSQFMKNESLLPQGSGISGTIIPTASD
GTKPKKDTSAEEAAKKEQQLKDMLQAIQAYIKDNNLEAKVSASDTPRGIAITLNDLFLFDLGKADLKPAAYPVLEKLATL
LPTLSNRVSIEGHTDNIPISSGSAYKDNWGLSFARSLSVLRYFTNTAKLNDEHLMATAYADTVPKMPNTSEENRAKNRRV
EIVVLREFQNDVQQPQPQKEAPPAAVAEAASAKAN
>lcl|NZ_CP026520.1_prot_WP_042233739.1_541 [locus_tag=PC41400_RS02795] [protein=flagellar motor protein] [protein_id=WP_042233739.1] [location=complement(618079..618879)] [gbkey=CDS]
MDLATLLGLILGLGALIGGYMWDGGHLTSLLVPSAMLIVFGGTFGAVAVSFPVATLKKIPKALGVAFKQTQRDTTATIDE
IVDMASIARREGVLALEQRAQEHPNPFLRDGLMMVVDGTDPELTRQILELEMDSIEHRNDGMSKMFEAAGGYSPTMGIIG
TVLGLIHVLGNLSDPSTLGPAIAVAFSATLYGVMFANVVYLPIGNKIKLRGKEQISEMELMLEGILALQAGENPQLIRKK
LASFVHEKNADKQSSGEAGAVLAEER
>lcl|NZ_CP026520.1_prot_WP_042233737.1_542 [gene=uvrB] [locus_tag=PC41400_RS02800] [protein=excinuclease ABC subunit UvrB] [protein_id=WP_042233737.1] [location=619170..621167] [gbkey=CDS]
MNEIIKSTNKFELVSDFSPQGDQPSAIQELVKSIQNGTRHQTLLGATGTGKTFTAAQVIAQLNRPTLIIAHNKTLAAQLC
SEFQEFFPNNAVSYFVSYYDYYQPEAYIASSDTYIEKDSSINEEIDKLRHSATSSLFERRDVIIIASVSCIYGLGSPAEY
GSLLLSLRTGMEKSRNEILHRLVDIQYQRNDLNFVRGTFRVRGDVIEIFPASRSEQAVRVELFGDEIERITEIDVLTGEI
LGERDHIAIFPASHFVTKEDTMKRALVNIERELEERLEELRGKGKLLEAQRLEQRTRYDIEMMAEMGFCSGIENYSGPLT
FRERGATPYTLLDYFPDDMLVMVDESHVTLPQIRAMYNGDRARKEVLVEHGFRLPSALDNRPLQFEEFERKVNQIVYISA
TPGPYELEHTPEMVQQIIRPTGLLDPIIEVRPTKGQIDDLIGEIQDRIAKDERVLVTTLTKKMSEDLTDYFKEIGIKVRY
LHSDIKTLERMQILRDLRLGTFHVLVGINLLREGLDLPEVSLVAILDADKEGFLRAERSLIQTIGRAARNAHGRVIMYGD
KITDSMDKAIKETERRRTIQEAYNKEHGITPQTIAKKIRDVIEATKVAEEKTEYLAEVKTGKMSKKDRAAVIERLEAEMK
DAAKNLQFERAAELRDALLELKAEG
>lcl|NZ_CP026520.1_prot_WP_042233736.1_543 [gene=uvrA] [locus_tag=PC41400_RS02805] [protein=excinuclease ABC subunit UvrA] [protein_id=WP_042233736.1] [location=621642..624533] [gbkey=CDS]
MARENIVIKGARAHNLKNIDVTIPRDKFVVLTGLSGSGKSSLAFDTIYAEGQRRYVESLSAYARQFLGQMDKPDVDSIEG
LSPAISIDQKTTSRNPRSTVGTVTEIYDYLRLLFARIGKPHCPVHGIEITSQTVEQMVDRVMEYPERTKLQILAPLVSGR
KGEHTKLLADIQKQGFVRVRVNGEIMDLSDKITLEKNKKHNIEVVVDRIVVKDDIQSRLADSLETALKLSEGRIIVDIID
QEELLFSQNLACPECGFSIDELSPRMFSFNSPFGACPECDGLGSQMIVDPDLLVSDPNKTIDQGAFEAWAGSTSTYYPQF
LAAVCSHYQIPMDVPVSELSQEQFKIILYGTDGQRVRFRYQNEFGTVKDAYVPFEGIVRNLERRYRDTFSDGIREHIEAY
MSSKPCPKCKGQRLKPETLAVTVGGTNIAHATELSIGDAQNWFEDLDGGLSEKERTIAHLILKEIKARLGFLVNVGLDYL
TLHRAAGTLSGGEAQRIRLATQIGSSLMGVLYILDEPSIGLHQRDNTRLIQTLEHMRNLGNTLIVVEHDEDTMMASDYII
DIGPGAGIHGGQIIAQGTPEEIMNNDNSLTGQYLSGKKFIPIPAERRKPNGKWLEIKGAKENNLRNVSVKFPLGVFTCVT
GVSGSGKSTLINEILYKTLARDLNKAKVRPGVYREMKGLEHVDKVIDIDQSPIGRTPRSNPATYTGVFDDIRDLFASTTE
AKIRGYKKGRFSFNVKGGRCEACRGDGIIKIEMHFLPDVYVPCEVCKGKRYNRETLEVNYKGKSISDILDMTIEDGCEFF
ENIPRIHRKLYTLLDVGLGYMKLGQPGTTLSGGEAQRVKLAAELYRRSTGKTIYILDEPTTGLHIHDIDRLLKVLHRLVE
NGDSVLVIEHNLDVIKTADYLIDLGPEGGSRGGEIVAAGTPEAVVKVEGSYTGQYLKPILDRDRKRSDDYAERAKELLED
AVS
>lcl|NZ_CP026520.1_prot_WP_042233734.1_544 [locus_tag=PC41400_RS02810] [protein=ferredoxin--nitrite reductase] [protein_id=WP_042233734.1] [location=complement(624806..626422)] [gbkey=CDS]
MAYQPIWAQDTSKLNKMELVKLQKDGLDVIRTIIEKYALEGYDSIPEDDMNLFKWAGVYQQKPKDGHFMMRVRINSGVLS
SAQARAIAGIAEDYGRNLIDVTTRQAIQFHWLTVESLPDIFKRLEEVGLYSYEACGDCPRTIVGNPLAGIDPDELMDTTA
LVNEVNDFFLMNKDFSNLPRKYKMSISSNVYNSAHAEINCLGFTPAVKVIDGEEVLGFHAWVGGGLSAKPHLAKQLDLFV
RPEEVLKVATGVTVIFRDFGYREKRHHARLKFLVADWGPEKFLEKLIELIGPMPTRGDEKVAGWTAAYFDGVHPQKQEGY
SYIGLNVPVGRLNAEELRELARISDQYGQGTIRTTMSQNIILSDIPNDRVDDVLQETVLQRLTPFPKRFMSRTVSCTGNE
FCNLAIVETKERARRVAEYLDANVDLDEEVRIHFIGCPNACGQKQIADIGLQGTLVKTPDGPVDAFDIAVGGILGPGAQF
NTGLKGRVKGDDVGPVLAELIRFYKENRSPEETFHQFFLRVGAPAFQEKLTEILTVAS
>lcl|NZ_CP026520.1_prot_WP_042233732.1_545 [locus_tag=PC41400_RS02815] [protein=hypothetical protein] [protein_id=WP_042233732.1] [location=626726..626926] [gbkey=CDS]
MLFAEEAAQASSFSGFDPFVIIFTILIAIGLVRLFAAKRKNVFAIAFALVSLVVFLFMDFVMIKGW
>lcl|NZ_CP026520.1_prot_WP_042233730.1_546 [locus_tag=PC41400_RS02820] [protein=sugar phosphate isomerase/epimerase] [protein_id=WP_042233730.1] [location=complement(627005..627973)] [gbkey=CDS]
MKLGVFAVLFGQKPLEEALDYIAEKGLEAVEIGTGGYPGNAHCKPEELLADESKLKAFKHAFDSRGLTISALSCHGNPLH
PQKAIAKQFHDDFIRTLELAEKLEVPVVNGFSGCPGDHEDAKYPNWPVAPWPNDFQEILDWQWTNKVIPYWTETGKFASD
RGVKIALELHGGFSVHTPGTLLRLREAAGEVIGANLDPSHMWWQGIDPVQAVQILGRENAIHHFHAKDTSIDPINVNKHG
ITDMQSYSLMLDRAWQFRTVGYGHDMKTWADIISALRLVGYDYVVSIEHEDGLMSVNEGFTKAVENLQQILIKEPLGDMW
WV
>lcl|NZ_CP026520.1_prot_WP_042233728.1_547 [locus_tag=PC41400_RS02825] [protein=MBOAT family O-acyltransferase] [protein_id=WP_042233728.1] [location=complement(628127..629278)] [gbkey=CDS]
MFYLNMNALLAMCGMVFVLMATRKWPQNKRILMLLFNLGFLFLWSGKLFAFYIAYTLLNYIFFVYLCRTPVWRKFAFALF
ILANIGAVILFHMLGKQGSASLLADGVMTLGIMFNVLKIIDALYFAYFFGKDGQAKVIDYVNYILFVPTFTSGPILKFRD
FMADVKVPYKVDASQFEAGVKRIILGLFKKVVVVVWMTDLFNKVMDQELHTHQSLFLMIWFYLMIYMDFSGYSDIAVGFG
RLMGITIPENFKKPFLSPTLTQFWRNWHATLGDWFRDHIFIFFSRKTPSKLMTAGLSIFIMTLMGLWHGFSWLFLIYGVY
HGVIIAIENLLGKTTVIRRKVSKPYFYMRCLLTQALVVAAIIINLNNPDVVLRVYKGLFHLPF
>lcl|NZ_CP026520.1_prot_WP_042233726.1_548 [locus_tag=PC41400_RS02830] [protein=hypothetical protein] [protein_id=WP_042233726.1] [location=complement(629431..630336)] [gbkey=CDS]
MNILAFFKKNSFVLGLLLIFVVAEALVMWWNPLETSRRFGKDDFTKTLYHHDWSRSGAVFYGNSSVTGAYIEEKSAAKLV
EMGLSYGKLTDLQQILQRGLYTVKDELVIGIDVHTLLDKMETDYRFPWNKPWYKPYVYAYRADFLDAGKELVRNLAFQGS
LAYEPRWIDKELYHGYATEEAMTKSWARYEKDYNWMTLKDMQDNLDALQWILGYSREQGIPLRVVWMPFRQDHVKPPYAE
AMKQEVNARLKAADVPVLDLMDRYAPKYFHDLVHLNREEGAPLFTKEVDEWLLSLRKPSKS
>lcl|NZ_CP026520.1_prot_WP_042233724.1_549 [locus_tag=PC41400_RS02835] [protein=hypothetical protein] [protein_id=WP_042233724.1] [location=complement(630348..631541)] [gbkey=CDS]
MECKISLERLSESQHGQVKYASAFVDEHIQEISLEDGHLRIVHTSPNGDEAIRERVERLIERFSKGDFGFKENVLFRNEV
SVPYNGDIIRELVESKSIKVLEPGLFIFREPFSSLMRFLDYSFVTKIANRFQVKEESYPAVIHSETLNKTNHFTSFPEHI
HFLTHLREDLDVIESFSDSIRATGGWKDELPLDLNATMPKPKFTMNPSTCYHCYEGLQGETLEGDGIVVTAVAKCHRFES
KNHTDFGRLLDFSMREIIFVGKPDFVKENRLKSIEYIKELAVEWEVDSLLEIANDPFFTNDFQVKASFQRNQEMKYELRL
TIPYLDKSIACSSVNFHSNTFGNAFDIKVGKRAAVTGCVGFGIERWAFAFLAQYGMNEADWPASFREQYHAWQQKSL
>lcl|NZ_CP026520.1_prot_WP_042233722.1_550 [locus_tag=PC41400_RS02840] [protein=acyl carrier protein] [protein_id=WP_042233722.1] [location=complement(631546..631797)] [gbkey=CDS]
MEEIMKQLEEHIRSRYEIEEDDDDFGVDVHLFDYGFIDSMGATALIAHIEQTYGIQVTNQDLMLYPMNTVREIAQFISTK
TGR
>lcl|NZ_CP026520.1_prot_WP_042233720.1_551 [locus_tag=PC41400_RS02845] [protein=VCBS repeat-containing protein] [protein_id=WP_042233720.1] [location=complement(631921..635142)] [gbkey=CDS]
MARIIYKNRGMNLSKKRTAIAVFCLAALLLAVGAAAFLTGKKEQVRLLYATGGDRYDTAAFEQFRQSLAANLELERKPMD
GLSARKLQSYDAVVPDPELANTPDNGYIQKLQDYVQNGGHLLLDNAFASKFPADFLGAYQTQLIVPPQAKPQFQYPAVPV
NAQGMQEVIKLFSDNFFKRETMDNLPGFYWGYGLDPSTAQPLVTLNGQALVSVNKVGKGTVLLSSTFLPNRYFITGYDLQ
SGMDPAQGFGSLAAKYDQQRSLKPGTAYFDKKDLKLEPYFNFAFSAANAQFRTEFVSYVSKETFGYSVTKVLGPYGRPAM
SYQNHFEAMPAFKDKEAIQWAELTKKYNQIPSFSLVRSAFYWGQWKESVSVHLNGGSNEKPQFSGESPASYYSSGMHLQS
EEGAIRLAEYPDYKSLADPVEGPHRLVPAFADLTGDGKEDLLAGSSDGYVYVYPNLGPKPEAYERDPAPGGLKLPDVYGK
PEKLLLTSGQPLKTGPHASVHAVDWNADGKIDLLVSDATGAVQIALGQGAGRFAPPTALQDPSGPIKVPGPAIAAAGDLG
GQGFPDLVVGDADGKVRVYRAPQAGALRLSAGAVVADVGAKYAAPSVRDMDGDGRADLVVGSNEGDLRVFRQESGGTFTA
QGSVGGATLNQMGTNALVGGHNSVPVWHDVNHDGVVDLIVGQLEFGSPVPVDSPSFPYKAELQEFLQYTKQNHLELYPHI
YVHNFVSDAQEKKEIALHKQAFENLGLPWTMPGTNQHTWRINFPDRLQTLRNENEQGIWFNFGFTPSHTPTEPRLGTDYI
WGLPFLLTKDKSGSDQPMLIYTPAPVLRKTGESSTVDIYDSYIKRDMPIDYFEHIEYHFPARVGELEEFASFLDDIRNQY
DYNFMTETQMARSFLTTLKTQVTVKRTWADYIMDEVKDRISGKGLHLSLTIGADTSAVPKQAAEYANTPGVVVEKGEKYA
PYSLAAKSDVYTERNGKLYLGLAREAQLHIGKPAEEAHIVRANVPFELDTRESGTWKLNLQAAGLQQIKIYSPSGLTIPG
NPADIKLEQDAATGMYTLTRYGDKTTISLSLKK
>lcl|NZ_CP026520.1_prot_WP_042233717.1_552 [locus_tag=PC41400_RS02850] [protein=methyl-accepting chemotaxis protein] [protein_id=WP_042233717.1] [location=635322..637466] [gbkey=CDS]
MSNKWSEKAAGVWQSAVKKIQNLAPNKSAAGGPSGPGGTGTSVPAMKAVGTKLFVIFFLSILVAVLVVGTLSYQTSSSVV
EEEVSKAAQQTVIQTGEKLDFLFRTYEKMSLDIMIDKAVGASVDQLARPNISDSDIAIARMNLDSNLKKYSQTDENLAGI
YIIPMTEKGVSTGYGQEKLIQENAALIKNVKEADGKAVWFDSKINGYFTQLSFPTFGVARVMKNDYGHSNYMVLLEIKTD
YLYNQLSSIRVGQTGRIEAMNTKQQLMVTEKADHIGQTVEVKLEGELQDQGRTKADVNGEQNLVVYTKSKQTGWLIYGFA
PLDEITEKANGIFALTVWVAIISALLAIVVGWLMSRMIGKPLVELGHLMNEGEKGNLAVRTLFTQRQDEIGQVGKSFNGM
MSKITDLVNQTNRSADSVLETARELTDASKKTALSAREIATATEDIAGGATSLAVAAENGNDLTTKISEQMKYVIEKNIR
MSIATSEVQQMSERGIEQMDELTQKTLQSEQMTHSLVDRVDTLKQSTSSIRKILDVLTKITAQTNILSLNASIEAARAGA
AGKGFMVVAGEIRNLADQSRESIVNVAHITEEIQREMDATVEALSAAYPLFREQTESVKEANQMFSSVKNEMDNLLIQLD
EVSTSINELNESQLTLADAMTNVSAVAEQSSATSEEVASLSSEQQSVSEGLVHLSGKLEDVSSRLSESLTKFKV
>lcl|NZ_CP026520.1_prot_WP_042233715.1_553 [locus_tag=PC41400_RS02855] [protein=glycine--tRNA ligase] [protein_id=WP_042233715.1] [location=complement(637639..639024)] [gbkey=CDS]
MSVNMDQIVALAKHRGFIFPGSEIYGGLANTWDYGPLGVELKNNVKRAWWKKFVQESPYNVGLDAAILMNPQAWVASGHV
GNFNDPMIDCKQCKARHRADKIIENALDAKGIELVVDGLSFDQMKALMDEHEIACPDCGSRDFTDIRQFNLMFKTFQGVT
ESSSNVVYMRPETAQGIFVNFKNVQRTMRKKLPFGIGQIGKSFRNEITPGNFTFRTREFEQMELEFFCKPGEDMKWFEYW
RAYCRDWLLNLGVKEDSIRLRDHSDDELSHYSNATTDIEYRFPFGWGELWGIADRTDYDLKQHMEHSGEDFNYVDQEAGN
EKYVPYCIEPSLGADRVTLAFLIDAFEEQKLEGDDSRTVLHLHPALAPFKAAIFPLSKKLSEGAQAIFSDLAKSFMVDYD
DTGSIGKRYRRHDEIGTPFCITYDFESEQDGQVTVRDRDTMEQTRMPISELKAFIESKVQF
>lcl|NZ_CP026520.1_prot_WP_129112447.1_554 [locus_tag=PC41400_RS02860] [protein=hypothetical protein] [protein_id=WP_129112447.1] [location=639325..639576] [gbkey=CDS]
MAPGWRIPFLSARGSGLPFRWFLARASTIPGSLIRRKFRYYWSFTAASYAVGNSLPKLRQKINAEAARPGLKIFFMEKEG
NKY
>lcl|NZ_CP026520.1_prot_WP_042233712.1_555 [locus_tag=PC41400_RS02865] [protein=glycosyl hydrolase family 18 protein] [protein_id=WP_042233712.1] [location=639832..641904] [gbkey=CDS]
MNGQAKRGLAFLLVVLLLLVSLPLSALGASEWAPGTAYKKGDVVSYSGSLYECVQAHTALQGWEPPNVPALWKKSTAVPD
TQAPTAPTGLTAGTVTGNSVALSWTASTDNVGVSGYEVYNGTALAGTSTTTAYTVTGLQPNTSYTFTVKAKDAAGNVSAA
SAAVTVKTAAPDTQAPTAPTNVTVGNVTANSVALSWTASTDNVGVAGYDVYNGTALAGSTSLTSYTVTGLQANTAYTFTV
KAKDAAGNVSPASAAVTGTTTNQPAGNNEIIAYFPSWTSYNTPAFTPANVDATKISAINYAFLDICWNGRHGNPSNDPEN
PNAQTWPCQNGNGSANNAPNGSIVLGDPFVDDVGEGGGYENLKKLAQLKTKNPKLKLFFSIGGWTWSNQFSNTAADATAR
QNFATSAVSLLRQYNFDGIDIDWEYPNAIGVPCASGQTCQRAADKANYILLLQTLRQQLDTAGQKDGKQYYITIASGANS
TFVADAGGTSNWITQAANYLDWINIMTYDFHGPWDSSSNHVAPLYNDPSDPAGTPQFNVDSAVKIYLNKGVPASKITVGE
PFYGYGWNGCSAGAKGDGLYQSCSGSSNTGSDGSTFNFSYLQNKGYLTKDASGKYTVGGLGFTRYWNDVTKTPYLYNPTS
KIFITYEDEASIHAKNEYAKSKGLRGVMFWELEADSNKTLQTVVSNDLPH
>lcl|NZ_CP026520.1_prot_WP_042233710.1_556 [locus_tag=PC41400_RS02870] [protein=DUF3656 domain-containing protein] [protein_id=WP_042233710.1] [location=642238..644907] [gbkey=CDS]
MRSLRREDIELLAPAGDWDCLRAAVANGANAIFFGVEKFNARARAHNFKMDELSDIMSFLHLYGVKGFLTFNILVFEDEL
RDAKELIEACVDAGVDAVIVQDLGLVKMIRDISPDFPIHGSTQMTITSPEAVEFTKPFEIERVVLGRENNLKQIKTIGDQ
AKLPMEVFVHGALCVSYSGQCLTSEMWGGRSANRGECAQACRLPYDLIVDGEPKPMGDVAYLLSPKDLAAIELVPQLIEA
GVTSFKIEGRLKSPEYVANVVSKYRRAIDAYFDGKDAKPSKEDVRELQQSFSRGFTSGFLEGTNNKKLVDGTFPKSRGVF
LGRVKQVLRDAVICEIEAPLKRGDGIVFDAGDPTKKEEGGRVYDLRRKGVKMEGEAEGGLIEIVPGRNDVDLGRVRVGDR
IWKTSDPALDKRLRQTYETEKPYRTFPVHVRVTGAAGQPLKSWWSDPSSNHSVFVESEALLEAALKRPLGEEVLGEQLGR
LGGTIFELAGLDVQLSGDVIVPMRELNALRRRAVELLEAERQKPRRYMKREVDAYADEQLLQAPVSAEERRASGRADAAG
RQESAEAAPAHTPDAAGKSADGTGAAAGGEAAQPYGVRLTALCRGIEQVLAAVETDVDMIYADFEFIKQFPAAVEAAHAA
GKKIALATPRIHMPGETGYFRNILKLRPDAVLIRNTGAMYFFMKYRAENPEEYQPELIGDFSLNIANHKTASLFVDAGCS
MVTPSYDLNIQQMVDMLSRTDTSKMEMVIQQHMPMFHTEHCVYCTFMSEGTDFTNCGRPCEDHRASLQDRIGMSHPVRVD
EGCRNTVYNAIEQSGAEYLNHFLELGVRSYRVEFLEESPEKVKEVLDLYRRAIRGNITGTQVWKTLKATNQLGVTRGQLV
KWGRVDTME
>lcl|NZ_CP026520.1_prot_WP_042233708.1_557 [locus_tag=PC41400_RS02875] [protein=DUF4179 domain-containing protein] [protein_id=WP_042233708.1] [location=complement(644978..646303)] [gbkey=CDS]
MDEQDWLQKELRGTTDGEIPAEADLAIAQGIQRAKALRRRRNRTFLTAAASLVLLVSVFGTMIHVSPAFAAAIRQIPGME
IFVRIMGTQDKGLARAVDNEFFQPINISHTQNGETVTLNGVIADESRIVVFYSWNRDHSTRKSGPEVKLHPADGSYMPAV
IGFQHSPVPAEPVTSGSDYLDAMLVEGTPIPDDFIFEVKLAGDDGPVWKVPFKLDRELFMNRAETIPIGKSVVIQGQTVK
FNEAKLYPTRILLDVSYEESNTKQLFGLRDIHLVTEKGERLSSWGASNNSENKKILTFESSYFTQPKELYLEGSLVTALD
KDQLTVKLDVANEQIISPDNRLQLVKLTDMGDSYAVSLKLSGLDSSWDKMNYTTWSDWKDAHGNTVEVSNHTSSRTFREE
PESNESFESFTIPKKDNGGILTTTLHEWPNPLREPYRIRIK
>lcl|NZ_CP026520.1_prot_WP_241688779.1_558 [locus_tag=PC41400_RS02880] [protein=sigma-70 family RNA polymerase sigma factor] [protein_id=WP_241688779.1] [location=complement(646296..646802)] [gbkey=CDS]
MLVRAARNGDDQAFTRLMDEEKARLYRIAYAYLHSRPDALDALQETAFRAYRNIRKLNDHTLFSTWVTRILLNYCADERK
RRNRPVYPDYTAVKEVSGPKPDNLDLYMALDRLQPSEKQLIILKYMEDWTYPQIASLLQMPESTIKTRARKALDQLKSIL
GREDEAYG
>lcl|NZ_CP026520.1_prot_PC41400_RS02885_559 [locus_tag=PC41400_RS02885] [protein=ABC transporter ATP-binding protein] [pseudo=true] [partial=3'] [location=647064..>648335] [gbkey=CDS]
MLEVNGVSKLYENSRGVREINFSMQRGEIVGFLGPNGAGKTTTMRMITGYLNPTGGNILVDGLSMADHPRKARRKIGYLP
ETPPLYPEMTVRSYLKFVADLRDVPVRAQKGRVNEIIDKLGLNGRENQIIRSLSKGYKQRLGLAQAIVHKPDLLVLDEPT
SGLDPKQIIEIRSLIRELGENHTVLLSTHILPEVNTLCNRVLIINQGQIVLDEKPEMLGATMGDTFEVSLEVKGPREEII
SLLGSTAGIAAVRELADGAASAVVLGLAGAPAAASGAGDSADAEAEAVSGGAGDTLSEAAETAAVAPASPDAAAEPAGSS
GSAVQADGPGGPLSGAVEEPAAAGADASAAPQAKTGGQQTASPGGTAPDTVKLLVSSADSTDIREPLFFRLAEAGYPILE
MKRESLSLEDIFLKLTTDESLSGS
>lcl|NZ_CP026520.1_prot_WP_042233704.1_560 [locus_tag=PC41400_RS02890] [protein=ABC transporter permease subunit] [protein_id=WP_042233704.1] [location=648460..649176] [gbkey=CDS]
MRRTWAICSKELQMFFYSPTSYVAFAFYVLVSGIMFYLNFVMVQPSLIDIRYVVGNITFIYLFIIPLLTMRLVADEFRHG
TDELLLTSPAGIGEIVLGKYLAALIVQFLLTLCSLVYPLIMSAFGTLDQPVLWLSYLSMFLLGAAMMAIGLFASTLSNNQ
MVAGIAGFAFLLFFWLIDWMAEGFGGKFRDWIIQFSMSGRMANMQKGILQGADILFFVTLAALFLILGMQTLERKRWR
>lcl|NZ_CP026520.1_prot_WP_042233703.1_561 [locus_tag=PC41400_RS02895] [protein=GldG family protein] [protein_id=WP_042233703.1] [location=649182..650570] [gbkey=CDS]
MKKWLRGTNTAVISVAVIGIFILLTFFLHSLKDFQLDLSKNKMYTLSDQTLGTLKDLGKDIHVTAFTNPQEDPDGVLTRE
VTDMVQEYSKRTGKITYNQYDMLKEPSKVQQYGVQQSSLVFESGSQKKVITFQEMFTAGQGSGSYQFAGEEKMTQAVKSL
TSTEKHTVYFLSGHQEITLQEMTVLKSSLEGENYTVKDLNLYREGKIPDDADMLFLIGPQTDLNDKEAELIKQYLSGKGK
LYVSLGFSQEMASKWKNIDALMNTYGVKDQHAVAIEAKQSTLYDPLTIIPEYGDHDITTKLKNYNLLTMMSLAVALNAQE
EGDFKSTPILKTSAQAYGETDIAGLMKSQTNQDDNDVKGPLNLGYAVEDKDSKPKAVILGGSTFLMDQEINVQGNRDFAM
NSIGWLQEKKDQVTIRPRQNEAYQTAALTPGQAKTIFFVAIVFMPLIFLLLGGFIWWRRRKG
>lcl|NZ_CP026520.1_prot_WP_042233700.1_562 [locus_tag=PC41400_RS02900] [protein=DUF4340 domain-containing protein] [protein_id=WP_042233700.1] [location=650567..651487] [gbkey=CDS]
MKKFAPALLLILVFLGGLWYAWSQDFFMEKEAETSKPLVHVNVADVQSFSILSGDQQTELVRKDGEWSMTKPSAAPLNPN
LTEGWMDSFNLLTQERIVEDKASDLAKYGLDKPSSVYKVQLADGTSQEVKAGSPTPVEGYVYVQLGSSPAVYQVGESSLS
VLNKSPLDFVETSPVKFDYDQVKSVTFSWKDHKWSLTKNEPDKSAAQSKWKLPDREIEGAEATQVLDKLLFMKTEEMPKR
AADEAIKPGEWTLEVVRTVDGKEMKDTYSGRSDGKSVWIAKQAGDWAYAIPQAAMEEVAASNVKKG
>lcl|NZ_CP026520.1_prot_WP_241688780.1_563 [locus_tag=PC41400_RS02905] [protein=hypothetical protein] [protein_id=WP_241688780.1] [location=651715..651963] [gbkey=CDS]
MAISHAGEHRAVYKEEQLERVTLPELQSLGDLIRFEHTLAKRCAEEAESSTDVELRGVLEQLAVLRLSCARQLLAALRST
EL
>lcl|NZ_CP026520.1_prot_WP_042233697.1_564 [locus_tag=PC41400_RS30530] [protein=spore coat protein] [protein_id=WP_042233697.1] [location=652045..653289] [gbkey=CDS]
MKQELQTDQAGEKQWEAAERTMADFKRLASRYASAAWDARQPGIRALYRQLLQQTLDDQDKLYRLLAGRGDKKKVQNASG
EDVLQEIASHDRALSELEAAIRKDFAGIEATEAETGESPEAVFQVLREAETISDPLTYEEKELEIFGERPDYYSSLSSSS
VYDSSYGDVYYSEPSVSDEEESVSAEAAETSASASAALEQPAPAPAASAVPAPLSPSRFGGAPAPTAPEQSAPAPATSPV
PAPLSPSRFGGAPASAGPEQPAPAPTASTTPAPLSPSRYGGAPASAAPEQPAPAPAGSLARPYTAPRSPYGRPATGGAAP
YGTPATAAPPVAYGQPASPSGVPVAPSYGGFRNPAPAPASAREPYGQSRRYGQPYARQPQQSGPTPPAAPPAQQPYNSPY
RGEDGRPPSYRPQG
>lcl|NZ_CP026520.1_prot_WP_042233981.1_565 [locus_tag=PC41400_RS02915] [protein=hypothetical protein] [protein_id=WP_042233981.1] [location=complement(653373..653672)] [gbkey=CDS]
MVKKSEQGTCELCGRHEVKRTEHHLIPKEKGGTFLPTAMLCIPCHKQIHALFTNKQLERGLHTLDALRDDERMRAFLKWI
RKQPSASLPRIRKSRELRQ
>lcl|NZ_CP026520.1_prot_WP_042233695.1_566 [locus_tag=PC41400_RS02920] [protein=DUF6130 family protein] [protein_id=WP_042233695.1] [location=653816..654238] [gbkey=CDS]
MRKISGKRSFVLMLLLIFACSGALTACGFQNKSEQVGGSAVSGAAPTIDATLTQEGTSAVISYKVSNLKIAGDSMGAAPK
QGEGHLHIYVDGVQKAMLKTDAPVRLENIPPGKHTIKLSLQQNDHVPLNVEKVFDVTIKK
>lcl|NZ_CP026520.1_prot_WP_042233693.1_567 [locus_tag=PC41400_RS02925] [protein=heptaprenylglyceryl phosphate synthase] [protein_id=WP_042233693.1] [location=654530..655216] [gbkey=CDS]
MIVDSSTWKHVFKLDPDREISDEALERICLSGTDAILVGGSSGVTFDNTVDLLARIRGFEVPCILEVSDAEAIVPGFDLF
FIPMVLNAGDPQWILGRHHEALKEYGAIMDWNEILVEGYVILNGESTAAKVTAARTDLTERDVVAYARMAEHLLRLPIFY
VEYSGTFGDMELVRRTRSVLEGTRLFYGGGIDGPEKAKLAAEAADTIVVGNLIYEDLDRALETVRALI
>lcl|NZ_CP026520.1_prot_WP_042233692.1_568 [gene=pcrA] [locus_tag=PC41400_RS02930] [protein=DNA helicase PcrA] [protein_id=WP_042233692.1] [location=655266..657599] [gbkey=CDS]
MIDIQQAVQKLNPQQRKAVEATEGPLLIMAGAGSGKTRVLTHRIAYLIGAKRVAPWSILAITFTNKAAREMRERVEQLVG
PSANDIWVSTFHSMCVRILRRDIDRIGFTSNFSILDSGDQLSVIKTCMKELNIDTKKFEPKAVQAAISTAKNELVTPQRF
EQKIGDYFDGLVAKVYTLYQKKLKSNNSLDFDDLIMATIQLFTDVPEVLDFYQNKFRFIHVDEYQDTNRAQYMLCQMLAD
KSKNLCVVGDSDQSIYRWRGADITNILNFEEDYPNATTILLEQNYRSTSTILQAANKVIGNNTGRKAKNLWTDKGEGSKI
RVYQADSEHDEGYFIAGEINKNVGSAQKYGDHAILYRTNAQSRVIEEILIKSDIPYQIVGGVKFYDRKEIKDILAYLRLI
SNPDDDISLTRVVNVPKRGIGDTTLDKVADFAARRGISMFAMLDEVESLEISARTRNTLADFRDMIHNLHRMVDYLSVTE
LTEKVLEMSQYRMELQRENTLESKSRLENIDEFLSVTSEFETRNEDKSLVAFLTDLALIADIDTMDKDEGPQDSVVLMTM
HSAKGLEFPVVFIIGMEEGVFPHSRAATDNEELEEERRLAYVGITRAEENLFLSCARMRMLFGRTSTNPPSRFLKEIPQE
LMEIVSSGGGGSTFSGWSRSGGTSFGGRTSSGFGSSGSTSFGQSSASGSSWGSQTSSFGKTSPEASTVRSTPRTAPAATG
SQPGDFKMGDKVSHGKWGTGVIVAVKGKGDDTELQIAFPAPVGVKRLLAKFAPITKQ
>lcl|NZ_CP026520.1_prot_WP_042233690.1_569 [gene=ligA] [locus_tag=PC41400_RS02935] [protein=NAD-dependent DNA ligase LigA] [protein_id=WP_042233690.1] [location=657641..659659] [gbkey=CDS]
MADFTAEMKTLIEAIEQHNYNYYTLDRPVVTDAEYDKLYDRLVELERESGVTLPNSPTRRVGGELLKGFEPHRHLSRLWS
LDKTRDKDGLNAWLTRVLKLVGEYNKQNPDNPLPHPTFVVELKFDGLTLNLTYDKGILVQASTRGNGTVGEGILEQVKTI
RSVPLEIPYKDGTVEVQGEGLMFLSVLDQYNQTAAEPFKNARNAAAGALRNLNPQVTAKRKLSAFFYNVGYSDNLRFTDH
REMVEFLKENRFKVNPDIKFFDSIDDVMEELERIGSNRLSYDFLIDGAVVKVADMRTREVLGYTEKFPRWAIAFKFEAEE
VTTVLESVSWEVGRTGKITPVARVEPVDLAGVTVQNCTLNNIGDIERKNLKHALGSLVYIRRSNDVIPEILGKATEETDG
EEIVFPENCPACGTPLEQRGAHLFCPNRLGCRPQMIGRITHFASRDAMDIDTFSVMTAEQLYQEVGVRDPADLYGLTYDD
LIKLERFGEKKANKLLAAFEKSKECDLSSFLFALGIPNTGKTTTKVLAEHLGSLDKVKEASIEELIALPDVGGIVADSIY
SFFRDPLMQDSISRMLAAGVAPRAEEKPAAAADESHPLFGKTIVLTGTLSAMSRDEAADKLEALGAKVTGSVSKKTDIVI
AGESAGSKLAKAQSLGIRIIDDEQELLALLGE
>lcl|NZ_CP026520.1_prot_WP_042235344.1_570 [locus_tag=PC41400_RS02960] [protein=hypothetical protein] [protein_id=WP_042235344.1] [location=complement(665401..665916)] [gbkey=CDS]
MKKNTSSDAIIYTYTLLRRIPHHLLLLGVYYAFAAVWAVKALLTGSLLSLCLGYIFVQLLHMLAIRGYLHFVRKTESQGW
SFRFAHLWTGYIPAQYVMLRSFVRLHLHLLISGFAGILLLYPWFSAAAISNILFVHLWILGGRFYMFLMFRKQYQVGWLR
LRVNETSCYKQ
>lcl|NZ_CP026520.1_prot_WP_009671897.1_571 [locus_tag=PC41400_RS02965] [protein=sigma factor G inhibitor Gin] [protein_id=WP_009671897.1] [location=666187..666372] [gbkey=CDS]
MEDPVLVKCIVCGEEKQEGIQIASEFICDACEAEIVRTDVMDVKYSFFIRQLKQIFYKQNA
>lcl|NZ_CP026520.1_prot_WP_042235341.1_572 [locus_tag=PC41400_RS02970] [protein=aminotransferase class I/II-fold pyridoxal phosphate-dependent enzyme] [protein_id=WP_042235341.1] [location=666484..668007] [gbkey=CDS]
MPNRENAPLFEKMREHYERRPASFHVPGHKSGRGADREAAPFFGPLMALDYTEISGLDDLHQPEEAIREAQALAADCFGA
EETFFLVGGSTAGNLAMIMSLCGPGDILLVQRNAHKSVLHGLMLAGAHAVFISPQTDGESGIPASVRLSDVEDALSRYPE
AKGLLLTNPNYYGMGVNLRPYADLMHRHGLPLLVDEAHGAHYGFHPALPESALACGADAVVQSTHKMLTAMTMGAMLHLR
HDRIDRERIRMLLAMLQSSSPSYPIMASLDLARRQMHLHGEKLVAEGLKAVQRLRDGLENIYGFRQLKHRSAQTAYEHMD
PFKVTIYDGTGTLSGSELQGFLEEAGCFVELAADKAVLLLLTPASTIEDAERALGACEAISKEYLTNKQELPGISPNSMY
KPLLNSVSKPVPMNIPVTRDVKNSIQDSVCVMPIESSAGKRSAEMVIPYPPGIPLLYPGELIQEDTAAELKRLAESGARF
HGSREAAKGRLRVLTDTPEDTCNSNIE
>lcl|NZ_CP026520.1_prot_WP_042235339.1_573 [gene=tmk] [locus_tag=PC41400_RS02975] [protein=dTMP kinase] [protein_id=WP_042235339.1] [location=668054..668719] [gbkey=CDS]
MKELFLTFEGPDGSGKTTQLRRLAESLQALGYDTVVTREPGGTPISDHIRSLVLTPDHKEMADHTEVLLYAASRAQHVRE
FIVPALKANRIVLCDRFVDASLAYQAYGLGISFDEVAAINRFATGGLEPTRTYLLDVPVETSRQRLLARTQSAGGEGLDR
IEQKGEAYHTRVREGFMKILEQNPKRICLIDADRTQDEIAHDILQDCKQLLDRVFFPKPEV
>lcl|NZ_CP026520.1_prot_WP_009671881.1_574 [locus_tag=PC41400_RS02980] [protein=cyclic-di-AMP receptor] [protein_id=WP_009671881.1] [location=668739..669068] [gbkey=CDS]
MKLVIAVVQDKDSNRLSNALIKEGFRATKLASTGGFLRAGNTTFMIGIEDERVPEVFKVIKANCKVRDQLVTPVSPMGGT
TDSYIPFPVEVQVGGAAVFLLPVERFEHF
>lcl|NZ_CP026520.1_prot_WP_042235337.1_575 [locus_tag=PC41400_RS02985] [protein=YaaR family protein] [protein_id=WP_042235337.1] [location=669127..669573] [gbkey=CDS]
MKINPGWQPIGKNVKVPETAPTLQSPPKDFSAFMLKHDEKVTTEQLNRMLQQIGLQGDRLSRSMKLRELRQYKLLIKQFL
EETARQGVGLRETRGWDRRGRTKRYKLLEEVDGALLSMADDLLETEQGRLEILQRIGEIRGMLIHFLF
>lcl|NZ_CP026520.1_prot_WP_042235335.1_576 [gene=holB] [locus_tag=PC41400_RS02990] [protein=DNA polymerase III subunit delta'] [protein_id=WP_042235335.1] [location=669594..670571] [gbkey=CDS]
MSFQTIPGQETAKRILQSGLRQDRISHAYIFSGPVGTGRKQTARMLAKAIYCTNLKDDACGECLECRKVEHGNHPDLYQI
EPDGASIKIDQIRELQKQFAYRSAGSGTKIYILNQAEKMTVQAANSLLKFLEEPNSKVVAVLVTENGHALLPTIQSRSQW
ISFTPMPRAEMARVLLEEGHPAALVQPAIHLTAGIQAAKEMIQTNGFAEIRSTMIQLLRETLTRYPSSFVTLQQKWIKAG
LADHTALLFDLMILWLKDMVSLRIGRKENFVYIDQMDWMGPLSFKRETGEWVRLLARAVELQKRLRFNANPQLVVEKMLV
DIQGV
>lcl|NZ_CP026520.1_prot_WP_016424270.1_577 [locus_tag=PC41400_RS02995] [protein=stage 0 sporulation family protein] [protein_id=WP_016424270.1] [location=670573..671376] [gbkey=CDS]
MYSVVGVRFKKAGKVYYFDPIDLPVEQENAVIVETARGIEYGRVVIGKKTVQENDVVLPLKKVIRIANPTDAQLVEENKK
AAKDAFGICLNKIKDHQLKMKLVDVEYTFDRNKIIFYFTAEGRVDFRELVKDLASIFRTRIELRQIGVRDEAKMLGGIGP
CGRILCCSSFLGDFEPVSIKMAKDQNLSLNPTKISGLCGRLMCCLKFEHDNYESAKDSVPSVGTYVITSLGNGKVLSLNM
NEKTCKVQLYDIGKIMDLPFDDVAEQD
>lcl|NZ_CP026520.1_prot_WP_042235332.1_578 [gene=yabA] [locus_tag=PC41400_RS03000] [protein=DNA replication initiation control protein YabA] [protein_id=WP_042235332.1] [location=671413..671751] [gbkey=CDS]
MEKHEMFAQIDGLEEQIGKAYEELGDLKKTIISLLEDNQRLMMENQQLRKLLKQGDAEETDSEPEKGIEPVIGEGYDNLA
RLYYEGFHICNVYYGHLRTEGDCLFCLSFLNK
>lcl|NZ_CP026520.1_prot_WP_042235330.1_579 [locus_tag=PC41400_RS03005] [protein=tRNA1(Val) (adenine(37)-N6)-methyltransferase] [protein_id=WP_042235330.1] [location=671826..672623] [gbkey=CDS]
MQVALHETERIDDLLTHDLKIIQSEEVFAFSLDAVLLARFCTVPPRGKIVDLCSGNGVIPLLLSTRTPAAEIWGVEIQER
LAGMGKRNIELNGLQDRLHMVHADLKQIHRTLGASGYDLVTVNPPYLPVPAGEQNRNEHVAAARHELFCTLEDVVASSAK
LLKTGGKMAMVHRPNRLVDICTLMRQYRVEPKRVRFVHPRQGEEAMMVLIEGIRDGKADMRTLPPLFVHEEGGRYCGELH
AVFYGEKSELEAGGQKSPENERNRG
>lcl|NZ_CP026520.1_prot_WP_042235328.1_580 [gene=rsmI] [locus_tag=PC41400_RS03010] [protein=16S rRNA (cytidine(1402)-2'-O)-methyltransferase] [protein_id=WP_042235328.1] [location=672630..673706] [gbkey=CDS]
MNFNIQRSYIMKKDDQRITGKEGDASDTAALEVEDSPNMERQAGADELQKGTYGHDVISPTPEQVQSRVRSRKTEKSAGT
GMLYLVGTPIGNLEDMTYRAVRILKEADLIAAEDTRQTRKLLTHFDISGRLVSYHEHNKQASGPELVRLMSEGQTIALVS
DAGLPAISDPGHDLVRLAIEAGVTVVPIPGANAALSALIASGLPTDEFTFIGFLPRDKKDQQRELARLAPAQGTLLFYES
PHRVVRTLEQMLESWGDREICLARELTKRYEEFGRGRISGALAHLGEHPPQGEYVVLVEGRSAAETAEAEAAWWKELSLE
AHAAHYMEQGLSKKEAIKKTASDRGLPKRDVYNELLQN
>lcl|NZ_CP026520.1_prot_WP_009671874.1_581 [locus_tag=PC41400_RS03015] [protein=AbrB/MazE/SpoVT family DNA-binding domain-containing protein] [protein_id=WP_009671874.1] [location=complement(674064..674318)] [gbkey=CDS]
MMKSTGIVRKVDELGRVVIPIELRRTLGIGEKDALEIYVDGERIMLKKYEPACIFCGNAENVTYFKGKIVCHECLNEMPI
PVTK
>lcl|NZ_CP026520.1_prot_WP_042235326.1_582 [locus_tag=PC41400_RS03020] [protein=HD domain-containing protein] [protein_id=WP_042235326.1] [location=674659..675933] [gbkey=CDS]
MIEEKVFKDPVHKYIYVQDRTVWDLINTKEFQRLRRIRQLGTSFLTFHGAEHSRFSHSLGVYEITRKIISQFERNSYADW
PEEERLLCLCAALLHDVGHGPFSHSIEKVFGTNHENWSCRIILGDTEVNQVLSRVSPTFAHDVAGVICKTYRKEIVTSLV
SSQLDADRMDYLLRDAYFTGVNYGTFDLERILRVVRPYKGKIVVKESGMHAVEDYLMSRYQMYWQVYFHPVTRSAEILLG
KVLECAKALFEDGYPFTFMPEPILHLFQGKLSLQDYFLLDESLMQTVLMYWSLEKDPVLADLSGRFLHRRLFKFSAVEEW
DKELLTEFRDALEEAGFDADYYLELDFPSDQSYDVYRPDTGKEDKQPILLLDGQDKLTEISARSEIVKSISGIHMGQHRI
YYPAELFERGGGKAVALQRKLFDA
>lcl|NZ_CP026520.1_prot_WP_042235324.1_583 [locus_tag=PC41400_RS03025] [protein=TatD family hydrolase] [protein_id=WP_042235324.1] [location=675969..676736] [gbkey=CDS]
MLTDTHTHMNNEAFDEDRDEAIQRAWDNGVTRIVNVGFNRETIPSSIELAEKYDFIWTTVGWHPTDAIDMRPGDLEWIEE
LCKHEKVVAIGEIGLDYHWDTSPKDVQARVFREQIRLARKVGLPIVIHNREAHQDIVDILKEEKAAEVGGIMHCYSGSWE
TAKQCLDMNFHISFGGPITFKNAKQPKEVLKQVPLDRLLIETDAPYLTPHPFRGKRNETSYVRLVAEAAAELRGMSLEEL
AKVTTDNAVRLLGLK
>lcl|NZ_CP026520.1_prot_WP_042235322.1_584 [locus_tag=PC41400_RS03030] [protein=ABC transporter permease] [protein_id=WP_042235322.1] [location=complement(676719..677981)] [gbkey=CDS]
MNQEAAVHREGTAGAVRQLWKRRFDHYRQETSRYWAYVARSQFFGFLFLIFIVSSYYYAKILQQLPTSFPYIWIVLLVLV
PALAAGSVRTLLKNADRVFLLPLEPVMNVFFRSAFRYSLGIQAARSIILMLIAWPLYKHCTGTEGQPLWLMAIFVLLIKT
ANLLGSWQEGRFSSPGWRLISVGYRWVLSAAVVFALFGTGPLPGGFVLLAGLVVWAVVYKLGPGQRTMGVAWDYLIDNDR
KQQYRLYLFFSGFADVPQLPVRVKQRRWLAGITDRYSFRSSSLYLYLFTKTFVRSELLGIVVRITLVAGLAAAAVQSAPV
AAAVCLVAMFISLVQLTSLEQSHRYTFWLQTYPVDPRRKVAAVVRIISGAFTLQGIVLTLLMLLSGTGVYAAAPLAGLAL
GLALCATVMKRKFTASFEAE
>lcl|NZ_CP026520.1_prot_WP_042235320.1_585 [locus_tag=PC41400_RS03035] [protein=ABC transporter ATP-binding protein] [protein_id=WP_042235320.1] [location=complement(677978..678721)] [gbkey=CDS]
MDPILHISRLSGGYLPGKPILRDLEITVGQGEMVGLIGLNGAGKSTTIKHILGLLKPQEGSVRIRNRTLEEDPLGYRSSY
SFVPESPELYDDLTIEEHLELTAMAYDIPKEAFKERAGRLMREFQMEPKRKSFPGQLSKGMKQKVMIMNAFLVEPALYVI
DEPFLGLDPLAMRSLLDIMADFKSRGASFLICSHILATIERYCDRFVVLHQGRILAQGTLEELRQQTGLSDATLDELFYE
LVRRNPA
>lcl|NZ_CP026520.1_prot_WP_042235318.1_586 [locus_tag=PC41400_RS03040] [protein=3D domain-containing protein] [protein_id=WP_042235318.1] [location=679491..680612] [gbkey=CDS]
MGNISQSETHVKRSSSMSFALRWKHENLRMIILAAIISFALTFMFLVLLYGTAGKSVSLVVNGRETVVETRQWVLQRLLD
EQAITIGEHDQVSMPLDGKIQDGQKIVIDHASPVQLTADGQTQTVHTIAKTVDQVLQDSNILLGEHDKVTPAKDTELSAN
AAIEVVRVKKETEEISEPIAFSVVKKSDEDLLKGKEQVVQEGQEGVLVKKKEKIFENGKLVSEQVIDQAVKNASVDKIVA
VGTKNPVVTVAAASEPVKPKQVTKAGTDIDAKKILKNVTLTAYSSEEPGVGNRTASGTKPVQGQTIAVDPSVVPLGWWVY
IEGIGYRRAEDTGGAIKGQKMDVYFDSMNYAKTFGVKRGYTVYVIGPNKPTQD
>lcl|NZ_CP026520.1_prot_WP_042235316.1_587 [gene=rnmV] [locus_tag=PC41400_RS03045] [protein=ribonuclease M5] [protein_id=WP_042235316.1] [location=680719..681264] [gbkey=CDS]
MIKEIIVVEGKDDTTAIKRAVEADTIETGGSAINRAILDRIRLAQERRGVIVFTDPDHAGERIRKIISGKVSGCKHAFLP
QELAEKRGDIGVENATPDAIRSALANVKTEYEGAVSLLTWEDLIAAGLVVHPDAAARRLEVGKKLGIGYCNGKQFYKRCA
AFQISREEFEAAVEQLDRPDQ
>lcl|NZ_CP026520.1_prot_WP_042235313.1_588 [gene=rsmA] [locus_tag=PC41400_RS03050] [protein=16S rRNA (adenine(1518)-N(6)/adenine(1519)-N(6))-dimethyltransferase RsmA] [protein_id=WP_042235313.1] [location=681313..682233] [gbkey=CDS]
MKLNEQGSGPSFASEDVATPKRTKDILNRHNLVLKKSLGQNFLTDGNVLRNIISAAELTPEKGALEIGPGIGALTQQLAK
EAGRVVAVEIDQRLLPILRETLSPYPTAEVVHGDVLKLDLRQLIEEKFSGLSGISVVANLPYYITTPIIMKLLEEKLPLE
NIVVMIQKEVADRMAAKPGGKEYGSLSIAVQFYCEAELVTIVPRTVFVPQPNVDSAVIRLRLRKGPPVEVADEDFFFEVV
QASFAQRRKTIYNNLGARFFSKENKRDLEALLLEAGIEPSRRGETLSMEEYGRVAEALRNCPTCNK
>lcl|NZ_CP026520.1_prot_WP_042235311.1_589 [gene=yabG] [locus_tag=PC41400_RS03055] [protein=sporulation peptidase YabG] [protein_id=WP_042235311.1] [location=682296..683174] [gbkey=CDS]
MRQGDLATRKSYGNDIIFRIQEIDTTRAILRGVDFRLLADAPISDLSRVTAARDSDFPRVNLPRVGETIRSIRQFRQELY
QLNDNLVRNKLPENTQYFEMPGKILHLDGDPNYLRKSMNLYGELRIPAEGFYVNESQMAATLQRLLPQVKPDIVVVTGHD
GLLKNRKKLDVHQLDSYKNSLHFVRAVQVARQYQNNRDALTIVAGACQSHFEALLQAGANFASSPARVLIHALDPLCIAA
KASYTSIRETVSMADLAGVTVTGLDGVGGIETRGSYRIGVPRLVNLTAEATI
>lcl|NZ_CP026520.1_prot_WP_009671869.1_590 [locus_tag=PC41400_RS03060] [protein=Veg family protein] [protein_id=WP_009671869.1] [location=683558..683839] [gbkey=CDS]
MAKNALLEIKRSLEPHVGQKIMLKANGGRRKTIERFGVLEETYPSVFIVKLDQEQQSFKRVSYSYADILTESVEVTVCDD
DAAEELRITYIQH
>lcl|NZ_CP026520.1_prot_WP_042235309.1_591 [locus_tag=PC41400_RS03065] [protein=small, acid-soluble spore protein, alpha/beta type] [protein_id=WP_042235309.1] [location=684022..684207] [gbkey=CDS]
MSRRRGRRGVMSEDFKYELAKDLGFYETVEREGWGGIRSKDAGNMVKRALQIAEQTLKRRP
>lcl|NZ_CP026520.1_prot_WP_042235306.1_592 [gene=ispE] [locus_tag=PC41400_RS03070] [protein=4-(cytidine 5'-diphospho)-2-C-methyl-D-erythritol kinase] [protein_id=WP_042235306.1] [location=684564..685418] [gbkey=CDS]
MKIYERASAKINLSLDVLRKREDGYHEVEMVMTMVDLADRIEMQELPRDTIIISSQAGYIPLDEKNLAFQAARLIKDRYD
VRKGVYIHLDKKIPVAAGLAGGSSDAAATLRGLNRLWGLGIPTEELQKLGGELGSDVPFCVTGGTALATGRGEILHPVDN
PLQCWVILAKPPINVSTSEVYGKLNAAGINHHPDTKGVLEAIRHKRFDELCGSLGNVLEEVTLQLYPQVRQMKEVMERLG
ADGVLMSGSGPTVFGLVSKEAKVARIYNGLRGFCKDVYAVRMLP
>lcl|NZ_CP026520.1_prot_WP_009671872.1_593 [gene=purR] [locus_tag=PC41400_RS03075] [protein=pur operon repressor] [protein_id=WP_009671872.1] [location=685589..686413] [gbkey=CDS]
MKKLKRSARLVEMTQYLLFRPHTLIPLTTFAERYSSAKSSISEDLAIIKEVFEDEGLGELHTLAGAAGGVKFIPKVAKEA
SLSFVQDLCNQLEQPERVLPGGFLYMADIMGRPQVLNEIGKTFATAFAGRELDVVMTVETKGIPLAYATASFMNLPVVTV
RRDSNATEGSAVSINYVSGSSKRIQTMSLARRTLQEGARVLIIDDFKRAGGTIQGMVDLLGEFKATVGGVGVFIESNEVE
EELVKDYISLAKLTGVDSKTKQVTVVPGNYFDSE
>lcl|NZ_CP026520.1_prot_WP_009671871.1_594 [locus_tag=PC41400_RS03080] [protein=RidA family protein] [protein_id=WP_009671871.1] [location=686430..686816] [gbkey=CDS]
MRILSTNEAPAAIGPYSQAVQVGNLVFTSGQIPLNAEGQLVEGGVTEQTHQVFRNLQAVLKEAGATLQNVVKATVFIKDM
NQFGAINEVYASYFGEHKPARSTVEVARLPKDVLVEIELIAAIAVENV
>lcl|NZ_CP026520.1_prot_WP_009671859.1_595 [gene=spoVG] [locus_tag=PC41400_RS03085] [protein=septation regulator SpoVG] [protein_id=WP_009671859.1] [location=686931..687215] [gbkey=CDS]
MQITDVRLRRVNSEGRMKAIASITIDNEFVVHDIRVIDGNNGMFVAMPSKRTPDGEFRDIAHPISSTTREKIQAAVLSEY
DRAAAEEEVIEEGA
>lcl|NZ_CP026520.1_prot_WP_042235348.1_596 [gene=glmU] [locus_tag=PC41400_RS03090] [protein=bifunctional UDP-N-acetylglucosamine diphosphorylase/glucosamine-1-phosphate N-acetyltransferase GlmU] [protein_id=WP_042235348.1] [location=687331..688728] [gbkey=CDS]
MKLMAIVLAAGQGKRMKSKLYKVLHSVCGKPMVGHVLDTLEHIQTEKTIVIVGHGAEAVKGYLGDRAEYALQEQQLGTGH
AVLQAKDVIGAEDGVTFVVCGDTPLVRPETLEKMLLLHKEKGAAGTILTAEMTDATGYGRIIRGEGGRVDRIVEQKDCSP
EEAAVREINTGTYCFDNHKLFEALENVKNENAQGEYYLTDVVGILKNAGEIVEAYCTPDEAEAIGVNDRIALSEAETLMR
QRINRGHMVGGVTFIDPSQTYIEKDVVIGPDTILYPGTTLRGKTFIGSECVIGPNTEICDSTLGDEVTVKQSVLQEAKIG
SHANVGPFAYLRPGTDLGEHVKIGDFVEIKNAKLGEGSKVSHLSYVGDAVIGKNVNFGCGAITVNYDGFNKSLTEVGDDA
FIGSNVNLIAPVKVGDGAYVVAGSTITHNVEDNDLAIARERQTNKKGYADRIRSKMKAKKENKHS
>lcl|NZ_CP026520.1_prot_WP_042235304.1_597 [locus_tag=PC41400_RS03095] [protein=50S ribosomal protein L25/general stress protein Ctc] [protein_id=WP_042235304.1] [location=688810..689439] [gbkey=CDS]
MGVALKGNTRTNATKSEAKQLRRQGKVPGVVYGKTTDSLTVVLERKELVPLLRSHPNAVIDLNIEGVGKRPVLISGVQRD
SLSQDVIHIDFHQINMNEPVKATVAVEIVGEAAGVKEGGILQVETAQLDVRCLPSDLPDSVQVDVSNLGIGENILVSDLK
LPENVEVKTEAQDVVVTVLQPQKEVEETTADEPAAEAEPAAEEAAESKE
>lcl|NZ_CP026520.1_prot_WP_009671908.1_598 [locus_tag=PC41400_RS03100] [protein=hypothetical protein] [protein_id=WP_009671908.1] [location=complement(689555..689755)] [gbkey=CDS]
MYDNGQQLYQDDHFQDHLEAEVPVQVYYLNAHKDIGFVYLYSSHFIKINNTFYNRGMFTFVSRPGY
>lcl|NZ_CP026520.1_prot_WP_042235302.1_599 [gene=pth] [locus_tag=PC41400_RS03105] [protein=aminoacyl-tRNA hydrolase] [protein_id=WP_042235302.1] [location=689989..690558] [gbkey=CDS]
MKCFVGLGNPGKQYEHTRHNIGFMAIDRFAEKWGVSSFQNKCKGHLAEVRVNGTKAYLLKPMTYMNLSGESIRAFMDFYK
VDVSDLVILYDDMDTPFGNIRLRYQGSAGGHNGIKSVIAHTGTSTFNRIRMGINRPNPGYPIVDYVLSNFSKAETGALPA
VLDKTCEAMEHCLSEPFEKTMAVFNGSGS
>lcl|NZ_CP026520.1_prot_WP_009671878.1_600 [locus_tag=PC41400_RS03110] [protein=anti-sigma-F factor Fin family protein] [protein_id=WP_009671878.1] [location=690775..691005] [gbkey=CDS]
MSVKYICRHCQSSIGEISRDQVSEYQLGFHFLTPEERSDIIAYNMDGGVTVKVVCDYCKQALDANPELNLIGSPLQ
>lcl|NZ_CP026520.1_prot_WP_042235299.1_601 [gene=mfd] [locus_tag=PC41400_RS03115] [protein=transcription-repair coupling factor] [protein_id=WP_042235299.1] [location=691133..694654] [gbkey=CDS]
MQALIQAFSTDRDFQSVITGIRSGMKEQLIAGLTGSSRQVMLAALMQQLERPLLIVTHNMFSAQKIFEDLSECLPEDGVL
LYPAQELLAAEAAIASPEMLAQRIDVLTKLAGGYRGAVIVPYAGFRRLLPTKETFKEAQLTVRLGDTIDLEEFLTRLTEL
GYERVERVEAKGEMSVRGGIVDLFPLTSSTAVRIELFDVDVDSIRTFDVSDQRSIEKLDSFVIQPCREVLLNRKRFQSAA
QHAYELLQTQLAKMTDRQAKDRLLEGIGHDIELLREGNTFQGLFKYVSLLYPERQTLSDYMPEDTVLLVDEPARLVETSK
QLEREEAEWVTSSLQDGKALPALTLSKQLETLMQRRTYPTLYMSLFLRQVPQTQPQNIVNFMSRVMQNFHGQMNLLKAEM
ERWKKTGSRVIILANGEERMDRVRRVLGDYQIDEPEIMNGNLQTGFELPSIHLVVITEGEMFTQKQRKARKVDKKIENAE
RIKSYQELKVGDYVVHVNHGIGKYVGIGTLEVGGIHKDYLHIVYAGGDKLSVPIDQIDLIQKYVGSEEKEPKIYKLGGAD
WNRVKNKVRTSVKDIADDLIKLYADRQATTGYGFSSDSAYQQEFESMFPYDETTDQLRAIGEIKKDMEKPRPMDRLLCGD
VGYGKTEVAIRAAFKAAIDGKQVAVLVPTTILAQQHYETFRERFSGYPFNIQVLSRFRSKKEQNETTKGVKAGTVDIVIG
THRLLSQDLKFKDLGLLIVDEEQRFGVSHKEKLKRLKTNVDVLTLTATPIPRTLHMSMLGVRDLSVIETPPENRFPVQTY
VLEYSTTLVREAIERELAREGQVYFLFNRVQGINQMAEQISMLVPEARVTVAHGQMSEQELEKTILDFLDGEFDVLVSTS
IIETGVDIPNVNTLIVHDADKMGLSQLYQLRGRVGRSNRIAYAYFTYQRDKVLNEVAEKRLQAIKEFTELGSGFKIAMRD
LSIRGAGNLLGAEQHGFIASVGFDLYSQMLSEEIAKRKLELDGEQAEPEKEWITQLDLHLDAYLPSDYIYDSMQKIEIYK
KVANIRSFAEAEDLDEELVDRFGDLPPAVSNLLAVARLKVYGAEYKIESISQKGDDYQIKISPDETERLDGQKLFTISNG
FDGRVKLIAGPQIHVHIRGKGLSPEDTIALVERFLLQYREALKTKGELQNATQ
>lcl|NZ_CP026520.1_prot_WP_042235297.1_602 [locus_tag=PC41400_RS03120] [protein=peptidylprolyl isomerase] [protein_id=WP_042235297.1] [location=694641..695801] [gbkey=CDS]
MQHNKSGWAKSKWLLLAVLVLVLAVASGCGKKDEGKAGTPEEVIATYKDGGKVTRGEFDKLLSFNRLLYPEYAQFETDPA
YQQSMLEQFIMLKVLSGRASDDVKKEADKQVTDQVTQLKTFVGAQEGGLEKKLKDAKLEEKDIEDLLRLSVYSTSYLEKQ
VTDDEIKAEYEKAKKADPNAYDVATVRHILIGLNDPNTNKELRTKEEALKRAKEVQQKLKNGGDFDALAKEYSDDPGSKD
KGGKYENAKVGQWVAGFKKAALELPLNTISDPVETDYGYHVMKVESRKTETVEDAKKDLASNLAQVKMQEFVTKELPGLI
ETNKLPKPTPTPAPSANPSPAPGGSTAPSGTEAPTGSPSPQTTPTPAPTDVPGATPPAASPSPAAK
>lcl|NZ_CP026520.1_prot_WP_042235295.1_603 [gene=spoVT] [locus_tag=PC41400_RS03125] [protein=stage V sporulation protein T] [protein_id=WP_042235295.1] [location=696034..696576] [gbkey=CDS]
MKATGIVRRIDDLGRVVIPKEIRRTLRIREGDPLEIFVDRDGEVILKKYSPIGELGDFAKEYAESLYESTNHITLITDRD
TIIAVAGGSKKEYMDKQIGQIVETTMENRKPTIESGSGTYEISKDNPETYSSFVIAPIVAGGDPIGSVILMSKDEEVTMG
NMESKMVETAAGFLAKQMEQ
>lcl|NZ_CP026520.1_prot_WP_042235293.1_604 [gene=spoVT] [locus_tag=PC41400_RS03130] [protein=stage V sporulation protein T] [protein_id=WP_042235293.1] [location=696734..697276] [gbkey=CDS]
MKATGIVRRIDDLGRVVIPKEIRRTMRIREGDPMEIFVNPDGEVILQKYSPVGELGDFALEYAESLNESTGYQAVIADRE
TFIAAAGRGRKEYDGKPVGEQLENALNGRQTILQEAAGIFELVKDAADAYGSFVIAPIIHNGDTIGAVLLVSKENDVVMG
KTERIMAETAASFLSKQMEV
>lcl|NZ_CP026520.1_prot_WP_042235291.1_605 [locus_tag=PC41400_RS03135] [protein=polysaccharide biosynthesis protein] [protein_id=WP_042235291.1] [location=697321..699123] [gbkey=CDS]
MSTEKGTQEQAVTVGSSRLLKGAALLAAAAVFSKLLGTLQKVPLQNIAGDGAFGIYNAVYPLYLLILFLATAGFPITVAK
FVQDRAAARDHAGARRVAAASALLLTGTGVLCFLLLYFGADVLAGWIGTAAAAPAIRSISYALLLVPVLSVLRGYFQGYQ
NMIPTAVSQAVEQTVRVLTMVLLLLYLTEQGAGDGDIAAGATFGSVTGAAGALLLLLYYWQRSKDRRTGRRRAAAKTSET
GDRPDSCNAPHSAGNSLDRSAVERPAAPGEAAQAEAAGPVGAKEPLGRLMASIGAYALPICLGSIMTPILNLIDAFTIPR
LLLGGGADPAEAMRQFGLYNHGLPLVQAVTMVASSLTAVLVPAIAAAKAADPAQLQIRTERTVRMTGLLGLAAAAGLASL
TVPVNVMLYQSEEASGTMALLALTALFGTLNIVLAGVLQGLGAVNAPALHLLAAAAVKIGGSLLLVPRWGIAGAAVSAVL
AFAAASALGLAEVRRLTGAPQRPRFSALAPLAAAVLMCAALGALQLVGGPLLGALAPGLGPRGQASVLALAGVALGAAVY
AFALLRFGAVNRAELEQLPGGGKLASALTKLRLFPQGERL
>lcl|NZ_CP026520.1_prot_WP_042235289.1_606 [gene=mazG] [locus_tag=PC41400_RS03140] [protein=nucleoside triphosphate pyrophosphohydrolase] [protein_id=WP_042235289.1] [location=699231..700712] [gbkey=CDS]
MTQGITVLGLGSGDENQLTLGVLRRLQQAPKLILRTDHHPVAAYLRAEGIRYETFDAIYEAKDRFDDTYEAIAASLLESA
RSQAVVYAVPGHPMVAEYTVQLLRAKCPEAGIPLEIVGGESFLDQAFLRFGFDPIEGMQLLDATDLRPGMLNPRLHTVIG
QVYDTFTASDVKLTLMELYPDDYTVFAGHSLGVAGEESVVRVPLYELDRIEGYGNLSLVWVPRSDRDELRNRTLERLHEI
VAILRSPEGCPWDREQTHASLRKNLIEEAYEVLETIDEDDPDHMCEELGDLLLQVMLHSQMEEETGAFTVYDVIAGLNEK
LIRRHPHVFGTEGAENAEEALANWQQMKDEEKRRKGIDPSKQSALAGVPKDLPALMKALKLQKKAAVVGFDWSEVGEVIA
KITEELGEVQEAMENGAAAERLDELGDLLFAVVNLSRFLKIDPEEALASTNRKFVRRFQYIEERLEARGKSLAEADLAEM
EELWQEAKKLQST
>lcl|NZ_CP026520.1_prot_WP_009671911.1_607 [locus_tag=PC41400_RS03145] [protein=HU family DNA-binding protein] [protein_id=WP_009671911.1] [location=700827..701099] [gbkey=CDS]
MNKTDLINNISTKSGLTKKDVETVLNSFLGEITDALSTGDKVQLIGFGTFETRKRSGRTGRNPQTGQEISIPESNVPAFK
AGNKLKEAVK
>lcl|NZ_CP026520.1_prot_WP_042235286.1_608 [locus_tag=PC41400_RS03150] [protein=RNA-binding S4 domain-containing protein] [protein_id=WP_042235286.1] [location=701099..701359] [gbkey=CDS]
MRLDKFLKVSRLIKRRTVAKDVSDQGRVWINGRDAKPSSTVKVGDELTIQFGQKKVTVRVEQLTESTRKEDAAHMYTLLR
EEANNG
>lcl|NZ_CP026520.1_prot_WP_009671862.1_609 [gene=yabP] [locus_tag=PC41400_RS03155] [protein=sporulation protein YabP] [protein_id=WP_009671862.1] [location=701480..701761] [gbkey=CDS]
MMEAGKTKRQEVKMLNRKLLEISGVKNVESFDSEEFLLETDCGFLLIKGQNLHIKNLSLEQGLVAIEGYVNALSYVDANS
PEKNKGFLGKLFK
>lcl|NZ_CP026520.1_prot_WP_042235284.1_610 [gene=yabQ] [locus_tag=PC41400_RS03160] [protein=spore cortex biosynthesis protein YabQ] [protein_id=WP_042235284.1] [location=701758..702339] [gbkey=CDS]
MTLNVQFLTLWFMFVSGMGLGMLYDAFRVVFVQLRLPRWTFPLADVVYWITATVCVFGTLVHSNQGQVRFFVFVGLIIGL
LVYMMTLSRTVRKVIAALIRAVKWLLRFFKRTFTLLFIKPAIGLYKLAWVLLGFLTALAMFLFRVVVQLVYPVWKLLLWL
TRPLYKHLRVPARVLRWLNFLKQLFQKWFRKGK
>lcl|NZ_CP026520.1_prot_WP_042235282.1_611 [locus_tag=PC41400_RS03165] [protein=septum formation initiator family protein] [protein_id=WP_042235282.1] [location=702396..702737] [gbkey=CDS]
MPVSPKSTQTQPKSNSKGTRRRFRIAAVILLGFLTWAGVTFWGQHGKISEKRAEMQQIKQEFTDEQKRKADLQQEIARLH
DPEYLEQVIRSELNYAREGETRFFIPTSKSQNP
>lcl|NZ_CP026520.1_prot_WP_016424266.1_612 [locus_tag=PC41400_RS03170] [protein=S1 domain-containing RNA-binding protein] [protein_id=WP_016424266.1] [location=702844..703326] [gbkey=CDS]
MAIEVGSKLEGRVTGITHFGAFVELEQGVTGLVHISEIADNYVKDVNDHLKLEDKVLVKVINVDKDGKIGLSIKQTIEKP
AEPARPERQGNGKFDSPRSGDRGGFGNRGDRGGRGGGFRNAPGKLSFEDKMNRFLKDSEERISSLKKQDSKRGGRGGRRD
>lcl|NZ_CP026520.1_prot_WP_042235278.1_613 [gene=spoIIE] [locus_tag=PC41400_RS03175] [protein=stage II sporulation protein E] [protein_id=WP_042235278.1] [location=703658..706150] [gbkey=CDS]
MEKRNGALSIGLQWAGAVQRTKSAAQGRVAENRLIQSFVAKKWSILLLLMAFLLGRAMILEELAPFSIAFFAVMYFQRRD
LLLWTGVFLFAGSLLAVQPQSGFILTEMVVFLLIQKGLEKYERSDISLAPVLVFITTLLVQMFAHLVSSDLSWYSLTMTG
VESVLSLVLTLIFVQAVPVFTLTRKNYQLKHEEIICLIILLASMMTGTVGWTLGPVTVEHVVSRYLILLFALVGGAPFGA
SVGVVTGLILSLANTNAIYQMSLLAFSGMLAGMLKEGRRMAVAFGMLLGSSILSIYIGTQGEVMNSTWESVAAVALFLLT
PKGVLQTLAKYVPGTQENLKSQQDYAKRVRDITAGRVEQFSQVFMQLARSFKQLTVESQPVPKEEEISHFMEAVSYQTCQ
MCPKRVRCWDQRFYQTYQFMTSMMTEIELNPNFSRKQIPIDWKQACTKPDQVLDVMKNQYAKYQNDMHWKKQISDSRQIV
ADQLSGVSQVMEDLAKEIKREGQELFLQEEQIRQALEDLGLSIHSIDVISLDEGNVEIEIIHQYTKGFDECRKIIAPLLS
EILGEHIAVKNEEMLPRGEGYSTVLFGSAKEFEVETGIAGAAKGGELLSGDSFSVAELGNGKFVVALSDGMGNGERARAE
SSAALTILQQLLQSGMDEKIAIKSVNSVLLLRSSDEVYATVDMALIDLYSAETTFMKIGSTPSFIKRGSEVIPVMANNLP
VGILQDIEVDLLSRSLQHGDLLIMMTDGIYDAPGHAVNKEMWMKRVIQEIDTDIPQDFADSLLERVMRCHHGEIIDDMTV
VAARVEKHQPEWATFRWPGVNRIERPKVVS
>lcl|NZ_CP026520.1_prot_WP_042235275.1_614 [locus_tag=PC41400_RS03180] [protein=VWA domain-containing protein] [protein_id=WP_042235275.1] [location=706419..707138] [gbkey=CDS]
MKQILLITDGCSNVGISPAAAAAEARAEGITVNVIGVVDQGELGLLGAEEIREIAKAGGGMSRIATSQQLSQTVQMMTRK
TVTQTVQQAVSRELQSILGSGDMTELPPAQRSEVVRVIDELAESGSLRVALLVDASASMKPKMAAVREAISDLLLSLKAR
SGTSELAVFHFPGRYDGDPDVEMDSGWTSNLANLDKMFYKLNMKGTTPTGPAMLKVAAYVSGKPIYSDDEDGMLRDYVV
>lcl|NZ_CP026520.1_prot_WP_042235273.1_615 [locus_tag=PC41400_RS03185] [protein=protein kinase] [protein_id=WP_042235273.1] [location=707122..708042] [gbkey=CDS]
MTTSYEPALPLGTTLKGIWNGRTYTVVRGLGSGANGQVLLVRRGERRYAMKIGYDPLDHQFEVNALQSLSKTDTSFSRFL
LDADDAEFEGRRLSFFIMPYIQGLSLSKYIRTYGKDWIYVIGSNLLKKLTELHRSGFIYGDLKSDNLVLREHGSVELIDF
GGMTAKGRAVKQFTEIFDRGYWCAGERVAEESYDLFSFAVLIIKTLDERDAFRPALQELPQNRNISMLEDIVKSSERMKG
LAPFLLRALNNRFASSEEALAVWREQAKKARGIKVPTERKDAPGIVLFLTISLIVLLGTIVYFYLD
>lcl|NZ_CP026520.1_prot_WP_053228800.1_616 [gene=tilS] [locus_tag=PC41400_RS03190] [protein=tRNA lysidine(34) synthetase TilS] [protein_id=WP_053228800.1] [location=708126..709643] [gbkey=CDS]
MEGKRQPKEANHSEPGSKGLDPGFRVRAAVESRIAEEKLLEPGDTVVVAVSGGPDSVALLHLLYGLAGSRNWRLIAAHVN
HRFRGEESDREAAFVASFAERLGLPCESAFIDVPAYIEETALNSQVAARQKRYEFLYETARKYGASKIALAHHADDQAET
ILMRILRGTGPGGLTGIPERRAEKKVELVRPMLRIYKKEVISYCEFLGLAYCRDSSNDSRKYFRNRIRLDAMPYLRQYND
QLPESLNRLGEIMTAENDYMEHEADRVYASVVKAEAGAVSLDRGDFVHLHVALQRRVIKLILNYLAFQEENVDYARLEAI
RTAIVQDSPSNLTWPIDGRIALIREYERICFRSETSARSGDAAFLHQVDLQQPFLPIPEAGMTFGFTAAERDGIGVADIP
VTCPDEAWFDLDALRMPLSVRNRQAGDRLDVFGLNGTKKVKDILMDAKIPPSRRDRLPLLMDAEGRVLWIPGVRRSRHAL
VGPGTVRWLHIRIEPPVTEGNGLGE
>lcl|NZ_CP026520.1_prot_WP_042235271.1_617 [gene=hpt] [locus_tag=PC41400_RS03195] [protein=hypoxanthine phosphoribosyltransferase] [protein_id=WP_042235271.1] [location=709684..710223] [gbkey=CDS]
MFNDIQEVFYSEEQIQAKVQELGTQISKDFEGRNPLVICVLKGAFIFMADLVKQVTIPLELDFMAVSSYGQSTKSSGVVK
IIKDLDVSVEGRHVLIVEDIIDSGLTLSYLIDVLERRSAQSVSVVTLFNKPARRSVDLQPDYAGFEIPDAFIVGYGLDYA
EKYRNLPYIGVLKPEIYSS
>lcl|NZ_CP026520.1_prot_WP_042235268.1_618 [gene=ftsH] [locus_tag=PC41400_RS03200] [protein=ATP-dependent zinc metalloprotease FtsH] [protein_id=WP_042235268.1] [location=710319..712295] [gbkey=CDS]
MNRFIRNTGFYLLIFLVTVGIVHFISTQNDQKEAITYNQFRQELTSKNVEKLQIKVDGGTYLINGQYKNPKAADKSAFVT
RGPFDSRVVELIETAGVQEEYLKMEGDSVWISFLTSIIPFVIIFVLFFFLMNQAQGGGGKVMNFGKSRARLYNEEKKRVT
FEDVAGADEEKQELIEVVEFLKDPRKFSAVGARIPKGVLLNGPPGTGKTLLARAVAGEAGVPFFSISGSDFVEMFVGVGA
SRVRDLFENAKKNAPCIIFIDEIDAVGRQRGAGLGGGHDEREQTLNQLLVEMDGFGANEGIIIVAATNRPDILDPALLRP
GRFDRQITVDRPDVKGREAVLKVHARNKPLAKDVRLDSIARYTTGFTGADLENLLNEAALIAARRNRKDISMAEIEEAFD
RVIVGTQKKSRVISEREKRTVAYHESGHAIIGYYAENADMVHKVTIVPRGRAGGYVMMLPKEGEDRMMQTKNELLDKVTG
LLGGRVAEEIFIGEIATGAYSDFQKATGIVRKMVMEFGMSDKLGPMQFGSSQGQVFLGRDIGHEQNYSDAIAYEIDQEMQ
LIIRECYDRAKQILNTHKDKVHLVAQTLLERETLDKEQIVELLEKGHISGNEDIKVTIQGQDDSVKPEPAPEIELKTETP
ERTKLNPDKPDDEGNNQA
>lcl|NZ_CP026520.1_prot_WP_009671902.1_619 [gene=nadA] [locus_tag=PC41400_RS03205] [protein=quinolinate synthase NadA] [protein_id=WP_009671902.1] [location=712509..713447] [gbkey=CDS]
MEALALERKAEQNRLLKERILELKKERNAIILAHYYQRDEIQEVADFRGDSFLLAQKAAETDADVIVFCGVHFMGESAKI
LAPTKTVIIPDERAGCPMADMVNVEGLRALKKEHPNAKVVAYINTSAEVKSETDICCTSANARKVIESIDAEEIIWVPDK
NLGDYVSKFTNKKMIIWEGYCNTHDMLTVKDVEEMKAAHPNAQFVVHPECRPEVVKLGDFVGSTTAIIKYCRESDCQEFI
VGTEDGTGYQLRLDSPNKTFHFATKYLVCPNMKVNNLKKVVRCLETMQPEIYVAPEVAEKAKLSLERMLQVK
>lcl|NZ_CP026520.1_prot_WP_042235266.1_620 [gene=nadB] [locus_tag=PC41400_RS03210] [protein=L-aspartate oxidase] [protein_id=WP_042235266.1] [location=713488..715119] [gbkey=CDS]
MIPRYLTDTDLTAIPHYKTDVIIIGAGIAGLFTALQASASQRVLLITKKSLLDSNTRYAQGGIAAVISDEDSPEYHLQDT
LVAGAGLCSRDAVDVLVHEGPQGVQDLIRMGTQFDLENGEIALTKEGAHSQRRILHANGDATGAEIVRALSENTKNNPSI
EIWDDHIVIDLITVDGECCGALVQKPDGQRVYVQGNATVLCTGGAGQLFRYTTNPDVATGDGVAIAYRAGADIRDMEFIQ
FHPTVLVYPGAPRFLISEAVRGEGAYLRNIKGERFMEKYHPQLELAPRDVVARAIVSEIEETGSSFIYLDITHEPEQMIR
HRFPTIYETCLSFGLDLTSDWIPVAPAAHYMMGGVKTDLHGETNIKRLFACGEVSSTGVHGANRLASNSLSEAIVFGRRI
IDRIRTLAPLPNGVPQAGEGTSARTELPKQAMVEKKLKLQKVMVRYAGVRRDAAGLQRGLDELKRHLPSFDSCLTRREDY
EFANLLTCALLTAEAALRREESRGGHYREDFPQRRDEQWLKHTVLNRECGLTEEWILNDGIKL
>lcl|NZ_CP026520.1_prot_WP_042235264.1_621 [gene=nadC] [locus_tag=PC41400_RS03215] [protein=carboxylating nicotinate-nucleotide diphosphorylase] [protein_id=WP_042235264.1] [location=715100..715969] [gbkey=CDS]
MMELNFKQIRQNIKLWLDEDIGTGDVTTMYTIPADQQSKGIIHLKESGIVAGLRVAQEVFTEVDENLVFSPQVKEGEFVS
KGTVIAIVTGNTRSILKGERLALNLLQRMSGIATRTRQYVNALEGLPTRLVDTRKTTPGHRMLEKYAVRVGGGFNHRYGL
YDAVMIKDNHIKGAGSITAAVRAARSRIPHTMKIEVEVENFEQLDEAIAVGADIIMLDNMKAADMAKAVSIVKSQAPHVL
IEGSGSVTLETIKSLAETGVHVISVGRLTYSVSALDISLDLNEQKGVSL
>lcl|NZ_CP026520.1_prot_WP_042235262.1_622 [locus_tag=PC41400_RS03220] [protein=type III pantothenate kinase] [protein_id=WP_042235262.1] [location=715966..716739] [gbkey=CDS]
MILVVDVGNTNIVLGIYQGTELLHHWRLGTNRSATVDEYGMMFHNLFQYAGIDIQQIEGVIISSVVPPLMFALEHLCLKY
VKRTPLIVGPGIKTGLNIRVDNPKEVGADRIVNAVAALELYGSPCIIVDFGTATTYDYIDKAGQLIGCAIAPGIGISTEA
LYQRAAKLPRIELIKPKSVVGRNTISAMQAGIIFGYVGQVDGIVDRVKQEFGIAPKVIATGGHAELIASESRTIDVVDPM
LTLEGLRFIYAKNAKQE
>lcl|NZ_CP026520.1_prot_WP_042235260.1_623 [gene=hslO] [locus_tag=PC41400_RS03225] [protein=Hsp33 family molecular chaperone HslO] [protein_id=WP_042235260.1] [location=716763..717638] [gbkey=CDS]
MSDYLIRGTALEGRVRAFAVLTTSITEELRQRHQTTPTATAALGRTAAAGLMMGAMLKGEEKLTLQVKGGGPIGSIVVDA
NSKGEVRGYVDDPNVDLPLNAKGKLDVAGAVGTDGFLYVTKDLGLKENYRGSIPIVSGELAEDFTYYFAKSEQTPSAVAL
GVLVDVDRTVKVSGGFIIQLLPGLTDDEITEIEKTLAQIPPITTLLDRGASLEEVLKEVLPSVEIMERSAVSFQCKCSRE
RVEQTIISLGRDELQSLIDEDGRAEVVCHFCNEAYQFEREELQTMLDSINK
>lcl|NZ_CP026520.1_prot_WP_042235258.1_624 [locus_tag=PC41400_RS03230] [protein=peptidylprolyl isomerase] [protein_id=WP_042235258.1] [location=717657..718607] [gbkey=CDS]
MHNVKVLWGAILALSIGVIVLLSLYLSDTGLGKKPEAGGRSGDTPADGHIVARITGREITQGELTEMLTKHYGAEMLNQM
LDRETLRLEAKEQGVDIGEAEISRELKRMQQGYENEQNFYDSMKEQLGMSKEDVKEDVLYKLLLEKLATHDVEIADDRIT
KYIQGHPEEFKQTSQIHLHKMVVSTLDQAKKATAAYRGGESFTTLARDRSLDDSGSQGGDLGWMEEDDPFQPAEIMKAAR
KLGVGQLSAPVKTAGGYALILITEKRMNTGPDEAAIRESVRKQLALQEAPPLNDYLLSLRQKWKATIADPGLNPNS
>lcl|NZ_CP026520.1_prot_WP_042235255.1_625 [gene=cysK] [locus_tag=PC41400_RS03235] [protein=cysteine synthase A] [protein_id=WP_042235255.1] [location=718721..719659] [gbkey=CDS]
MAKLVQSITQLIGNTPLVKLNRIVPEDSAEIYVKLEFQNPGASVKDRIAISMIETAEQEGKLKPGDTIVEPTSGNTGIGL
AMVAAAKGYRAILVMPETMSMERRNLLRAYGAELVLTPGAEGMKGAIRKAEELQAENPDYFIPQQFKNQANVKIHRETTG
PEIVEAIKGHDGKLDAFIAGIGTGGTITGTGEVLRENFPGIKIVAVEPSASPVLSGGKPGPHKIQGIGAGFVPDILNTEI
YDQIIPVDNEDAFETSRSVARQEGILGGISSGAAIFAALKVAKELGPGKRVIAVIPSNGERYLSTPLYQFDE
>lcl|NZ_CP026520.1_prot_WP_042235254.1_626 [locus_tag=PC41400_RS03240] [protein=anthranilate synthase component I family protein] [protein_id=WP_042235254.1] [location=719899..721497] [gbkey=CDS]
MSATTITLTTLADWTEWAEAGAYTSVPYITSIPWTQDISTWEHAWRAAGTHAFVLESGRGGRYTFLGLHPVAAICGSAAS
AEAAEQDGAVPDGWRSVPMQGAPLETVKRWMAPWRSPDVAGAPKFTGGCVGFWSYDVVRSMEKLPELAEDDLPLPEYSFA
RYDEVWVFDREEGRLYAAVHTPLPSEHSVSDLSRLYQAAVQRSESMLRLWHRLAGDTEAGAQRLQREAEWQALVSAGEDT
TLEAEAAEGASRAFPRESFIRAVERIQAYIGQGDVFQVNLSVRQSRPLQTSPEQIYEALRIVNPSPYMGLMRFPGFALVS
GSPELLVQLEGRHLRTRPIAGTRPRGRTPEEDARLADELLSNEKERAEHVMLVDLERNDLGRIARYGSVKVDEFFVIERY
SHVMHLVSEVTGELREGLDAFDVIAAAFPGGTITGAPKIRTMEIIEELEPVRRGPYTGSMGWIDYNGNMEFNILIRTLVA
VGGSGYIQAGAGIVIDSVPDREYEEALNKARALWKAIQFSEHSMSRAGTEVL
>lcl|NZ_CP026520.1_prot_WP_042235252.1_627 [gene=pabA] [locus_tag=PC41400_RS03245] [protein=aminodeoxychorismate/anthranilate synthase component II] [protein_id=WP_042235252.1] [location=721521..722108] [gbkey=CDS]
MILVIDNYDSFTYNLVQYLGELGQDIVVKRNDAIDLSGIEEFSPDHILISPGPCTPNEAGISLDVIRHFKGRIPILGVCL
GHQAIGQAFGGDVVRAGKLMHGKTSPIYHDGKTIFEDIPSPYTATRYHSLIVRKETLPDCLEISATSDEGEIMGLRHKEY
PIEGVQFHPESIITDHGLTLLRNFLGRTVPAGTGA
>lcl|NZ_CP026520.1_prot_WP_042235250.1_628 [gene=pabC] [locus_tag=PC41400_RS03250] [protein=aminodeoxychorismate lyase] [protein_id=WP_042235250.1] [location=722110..722988] [gbkey=CDS]
MKIWMNGNLVEEKEAVVSVYDHGFLYGMGLFETFRTYNGHPFLLKEHMERIREGCAELGIPFHTDETGMRARVEELLRAN
GLEDAYFRYAVSAGVDMLGLPGGAFYENPSEIIYVKSLPPRNADQPASRPLQLLNLRRNTPESGRRHKSFHYMNNILAKR
EMMNYPWAHGAEGLFLTDNGGYVAEGIVSNIFWIKGVVCFTPSLTTGILPGITRAFVLKLAEEEGLEISEGLFPWSELCE
ADEIFVTNSVQEIVSINELFDERGGSITVSRGMAGPVTSRLAQLYRECTRLL
>lcl|NZ_CP026520.1_prot_WP_042235248.1_629 [gene=folP] [locus_tag=PC41400_RS03255] [protein=dihydropteroate synthase] [protein_id=WP_042235248.1] [location=723010..723840] [gbkey=CDS]
MIRTFAAGETSFELGRRTLIMGILNVTPDSFSDGGSYTTVEHAVKHALSMVEEGADMIDIGGESTRPGHAPVSLREELER
VVPVIRELVRHVRVPISVDTYKAEVARQAIEAGAHIINDVWGFKADPDIAGVAAQAGCPVILMHNRPEAVYTDFLPEVLQ
DLRESIDLALSAGVKKTNIILDPGIGFAKSYEQNLWLMNRLEAIAELGYPVLLGTSRKSMIRTTLGLPPDDVIEGTAATT
ALGIAQGCDIIRVHDVHAMKRVAVMTDAIVRGPGSM
>lcl|NZ_CP026520.1_prot_WP_009671857.1_630 [gene=folB] [locus_tag=PC41400_RS03260] [protein=dihydroneopterin aldolase] [protein_id=WP_009671857.1] [location=723930..724298] [gbkey=CDS]
MDKLILTRMEFYGYHGVFAEENKLGQRFYVDVELHMPLDKPGRTDNIEDSINYAEAYAVIKEVVEKRTFKLIEALAEQVA
SSLLETYTSIEEVTVRVIKPHPPFDIHFAGVTVEISRKRAQS
>lcl|NZ_CP026520.1_prot_WP_042235246.1_631 [gene=folK] [locus_tag=PC41400_RS03265] [protein=2-amino-4-hydroxy-6-hydroxymethyldihydropteridine diphosphokinase] [protein_id=WP_042235246.1] [location=724295..724831] [gbkey=CDS]
MSPADITTAYIALGSNIGDRERYLAEALKKLDEHPHVQVLACSSIYETDPVGYVDQDAFLNMAAAVHTSLSPEGLLAVMQ
NIEQVLGRVRDIRFGPRTIDLDLLLYGEREIRTPDLIVPHPRMQERAFVLVPLADLFSHIPNTSFASIEKQLAKLDGKEG
VMLWKKMQWPSESGHSAS
>lcl|NZ_CP026520.1_prot_WP_042235244.1_632 [locus_tag=PC41400_RS03270] [protein=helix-turn-helix transcriptional regulator] [protein_id=WP_042235244.1] [location=724783..724995] [gbkey=CDS]
MEKNAVAQRIRAFRKLKGYTQNQLAELLGVSIAVLGSVERGTRKPDTRMIQKISNALGIEPEELLPPSGK
>lcl|NZ_CP026520.1_prot_WP_042235242.1_633 [gene=dusB] [locus_tag=PC41400_RS03275] [protein=tRNA dihydrouridine synthase DusB] [protein_id=WP_042235242.1] [location=725007..726017] [gbkey=CDS]
MLKIGDVVAPNNVVLAPMAGVCNPAFRLIAREFGCGLVCAEMVSDKAIVHGNSRTMEMLYVDEREKPLSLQIFGGDTETL
VQAAKYVDANTNADIIDINMGCPVPKITKCDAGARWLLDPSRIEEMVRTVVGSVNKPVTVKMRIGWDSEHIYAIENAQAV
ERGGGSAVSVHGRTREQLYTGKADWNIIRDVKQAVSIPVIGNGDVVTPEDAKRMIDETGVDGVMIGRGALGNPWMLYRTV
QYLTKGELPPEPEAEEKMRIATLHMDRLIKLKGENVAIKEMRKHMAWYLKGMKGAARLKDAIMEQTTRDGMVRILDEYAS
SFDNMSDNQEEQPALH
>lcl|NZ_CP026520.1_prot_WP_042235239.1_634 [gene=greA] [locus_tag=PC41400_RS03280] [protein=transcription elongation factor GreA] [protein_id=WP_042235239.1] [location=726165..726641] [gbkey=CDS]
MSEKEVLLTQEGLKKLEEELEHLKSVKRREVAERIKVAIGYGDISENSEYEDAKNEQAFIEGRVITLEKMLRNARIVNLD
EVDINTVSVGSIVTLKDLEFGDLVEYTIVGTAESDPFLNKISNESPVGKAILGKQKGAKVDVTVPAGVIQYEIMDIKK
>lcl|NZ_CP026520.1_prot_WP_042235237.1_635 [gene=lysS] [locus_tag=PC41400_RS03285] [protein=lysine--tRNA ligase] [protein_id=WP_042235237.1] [location=726846..728345] [gbkey=CDS]
MSQELELNELLQIRRDKLDELRGLGVDPFGDKYVRTHTAKQILDAYEETPKEELEQQTVEVSIAGRIMQKRAMGKASFAH
LQDITGKIQIYVRQDALGDTKYSAFDLLDIGDMVGVKGTVFKTKTGETSIKVQELEVLTKSLLPLPEKYHGLKDVETRYR
QRYVDLIMNQEVKETFILRSRIIQSMRRYLDGLGYLEVETPTLHAIAGGAAARPFITHHNALDMQLYMRIAIELHLKRLI
VGGLEKVYEIGRVYRNEGISTRHNPEFTMIELYEAYADYKDIMALTENMVAHIAQEVLGTTTVQYQGHEVNLAPSWKRIS
MVDAIKEVVGVDFSVHMSNEEAQRLAKEHKVPVEPHMTFGHIVNQFFETFVEETLIQPTFIYGHPLEISPLAKKNPEDPR
FTDRFELFIVAREHANAFTELNDPIDQRQRFEAQLLEREHGNDEAHEMDDDFIRALEYGMPPTGGLGIGIDRLVMLLTNA
PSIRDVLLFPHMREQRQGE
>lcl|NZ_CP026520.1_prot_WP_042232360.1_636 [locus_tag=PC41400_RS03350] [protein=MTH1187 family thiamine-binding protein] [protein_id=WP_042232360.1] [location=complement(734906..735214)] [gbkey=CDS]
MAIAEITVVPIGTSSTSLSEYVAHLQKVLDNQPLHYELTPMSTIIEGPLDQLMEVILKLHESPFENGAQRVSTSIKLDDR
RDKAASMQQKLQSVADKLEQLQ
>lcl|NZ_CP026520.1_prot_WP_042232358.1_637 [locus_tag=PC41400_RS03355] [protein=GMC family oxidoreductase] [protein_id=WP_042232358.1] [location=complement(735306..736865)] [gbkey=CDS]
MKWNTQSETVDVVIIGAGAAGGVLAQELSEAGLQVVVLEAGPLRDPQKDFASDELSMQCLGWQDTRLVDGRHPLTMGHNN
SGWGVGGGTQHFTGVFLRFHESDFKTRSIDGVGEDWPIHYKDLEPYYSRIEKEIAVSGPRHFPWGNFRGPYPYPERDPLS
PNAYLFQRGCESLGIRSVTAPLAILSAPFEGRPPCINRGFCQQGCMPNAKFSTLIQHIPKAVQAGATVFTDSRVTQIETG
PDGMATAVLFVKKGVEYRQKSKLVIVSAYAVETPRLLLNSANSRHPDGLANSSGWVGKALMTHSSHDVYAKFEEEVRLYK
GTPVLATTQEFYETDRSRGFARGYSLHAHGTRPLAFAQGIVKTSKSPVWGTPLREAMLDYNFYGRTTLVGEVLPQEHNCV
TLSGEKDTFGLPRAKVSFSFGENDIKLIEHAVSNMKSILQAAGGTPEFVVPDTAHLMGGCRMGNDPSRSVTNSFGQTHDI
PNLFVCDASLFVTSSGVNPTNTVMALAARTADYIVSEII
>lcl|NZ_CP026520.1_prot_WP_042232356.1_638 [locus_tag=PC41400_RS03360] [protein=gluconate 2-dehydrogenase subunit 3 family protein] [protein_id=WP_042232356.1] [location=complement(736862..737458)] [gbkey=CDS]
MRKMLKYPSFDVMKEQNHWDPHTRSIVSSRMRRRREYRLLTRLEAEQLRSVCSLLVDDDRSEILTYVLEHIDDTLCGQKE
EGQRGPDTPPQSELIRKGLEALETMSHGFHARPFQQLSPAEQKELLTNVAEVRLSLRGDWPAAVQKAFFIRLLTLTVEAY
YSHPQIWSEIGYAGPAYPRGYIRADRGHLDPWEAKPER
>lcl|NZ_CP026520.1_prot_WP_042232354.1_639 [locus_tag=PC41400_RS03365] [protein=hypothetical protein] [protein_id=WP_042232354.1] [location=complement(737464..737856)] [gbkey=CDS]
MIWNAWTLDRSVILFTAFAYLFIAAQVTLSHYRQNFHRKVMLIPVITAPLYFLVGLLAVLLNRSWLLCLFHIGMWLGILT
GLAGLYLHIRGVGKRVGSYAFRNFLIGPPVLMPLLYSALGILGLIAVYGR
>lcl|NZ_CP026520.1_prot_WP_042232352.1_640 [locus_tag=PC41400_RS03370] [protein=MraY family glycosyltransferase] [protein_id=WP_042232352.1] [location=738117..739064] [gbkey=CDS]
MHYLIPMLTSFILVALLVPVMKRIAVKAGFVDRPSTRKIHAAPIPLLGGVAIYIACAAAILMFSRNGNLSYAVLIGGTVL
MLTGLTDDWFKTRSKDFPVWPRLIIYIAVSAVPLLFGIQIIGVSNLHFSGMIFFPVWLVWLGTMAWIFGITNMINFIDGV
DGLASGIVTIASLALLVAALVKRQEDSALIAGILVGACLAFLAFNFYPAKIFMGDAGAIFLGYTIAVVAVDGAFKSATFI
SLLVPVLALGVPILDTVIVFTRRLLKGGGLHRADKLHTHHSLMQWGLSQTQTVSFLYLIGALFSLLSIIVVLMGV
>lcl|NZ_CP026520.1_prot_WP_042232349.1_641 [locus_tag=PC41400_RS03375] [protein=helix-turn-helix domain-containing protein] [protein_id=WP_042232349.1] [location=739186..740301] [gbkey=CDS]
MDWERLKEQLESIGKGPVQLIAKTPEEWETLTRNAGKNNGDRQAVKKSSMLIGKEVYFLLSREGNEIHTLSIPAGNLAGD
ERHLVEWLIESAQPEKRQSGTLLLEEEKKLSALREWLLQHAETDSQAEVPDTISSQAGLYSTKIPFLLYGDYSSNIRASY
RELKRLLETFFDAEIILIPLLDKEWLILAPDYLLTVGQEDKTDAQEESTEQMLDALASGLHDMLATEWVGESHLGVHYPI
QPAQALLAAMLSLRSTIMLGRLYNVTNNIHTPWHLQMEKLLHPIPDSEKHQFLEHVLKDYSMDAETQSTLESFFALDCNV
SETAKKLYIHRNTLLYRLDKFKNETGLDVRAFHDAVLVKLALVLYKVTKRK
>lcl|NZ_CP026520.1_prot_WP_042232346.1_642 [gene=ugpC] [locus_tag=PC41400_RS03380] [protein=sn-glycerol-3-phosphate ABC transporter ATP-binding protein UgpC] [protein_id=WP_042232346.1] [location=740403..741527] [gbkey=CDS]
MAGVRLNHIVKRYPGNQEATVKDFHLDIQDKEFLVLVGASGCGKSTTLRMIAGLEEISEGELYIGDRLVNDVAPKDRDIA
MVFQSYALYPHMNVYQNMAFGLKLRKFKKADIDARVREAAKILDIEHLLDRKPKALSGGQRQRVALGRAIVREPQVFLMD
EPLSNLDAKLRVQMRAEISKLHKRLGVTIIYVTHDQTEAMTMGSRIVVMDKGIIQQAASPEEIYNFPVNMFVAGFIGSPS
MNFVNGSLSDQGGELHFKAKGVDVVIPQGKAQVLRDKGYVGKDVILGIRPEDIHDEPVFLEASAKTVVNALIEVSENLGH
EMQVHLAGIGEGTVVARVDGRSGLKDGANVKLALDMNKVHFFDKETTNSILLNS
>lcl|NZ_CP026520.1_prot_WP_042232344.1_643 [gene=hprK] [locus_tag=PC41400_RS03385] [protein=HPr(Ser) kinase/phosphatase] [protein_id=WP_042232344.1] [location=741718..742659] [gbkey=CDS]
MGKKVRVSELVEHYNLEVICGEDGLKRTVTVADLYRPGLEMAGYFNYHPQERVQLLGRTEMSFFETLDETVRLHRAEQLC
ASEETPCIIVTRGLDIPAELVEEAKKHQLPVLRSQVATTILASRLTNFLENKLAPSTTIHGVLVDVYGVGMLITGGSGIG
KSETALELVKRGHRLVADDAVEIRQTADYMLYGNAPDLIRHLLEIRGVGIINVMTLFGAGAVRNVKKISVVARLETWQQD
KQYDRLGLDEETTRIIEADVPLITIPVRPGRNLAVIVEVAAMNYRLKRMGFNAALQFTNKLTETLAEDLDDMD
>lcl|NZ_CP026520.1_prot_WP_042232342.1_644 [gene=lgt] [locus_tag=PC41400_RS03390] [protein=prolipoprotein diacylglyceryl transferase] [protein_id=WP_042232342.1] [location=742710..743645] [gbkey=CDS]
MLNPVAFKLGPISVHWYGIILGTAALIGLLLVIREGKRFKLQPEFFMDLLLIGVPSAIVAARIYYVAFQWENYKNNLGEI
IAIWHGGIAIYGALIGAIIAALIYVRRKGYSFWRIADICAPGLIVGQMIGRWGNFVNQEAHGGPVGESFLRNTLHLPDWI
VNQMNIEGIFYHPTFLYESVWNFVGLLLLFWLRRRPFLRAGELFMSYFIWYSLGRFFIEGLRTDSLDFTGPAWLASLLNG
LWSPMSLLFGEAGTMTYGGNIRISQLLAVLIIIAAIIIIVLRRRMGYAKTKYSDPIEKIGGAPASANTSQS
>lcl|NZ_CP026520.1_prot_WP_042232340.1_645 [gene=ppaX] [locus_tag=PC41400_RS03395] [protein=pyrophosphatase PpaX] [protein_id=WP_042232340.1] [location=743679..744332] [gbkey=CDS]
MKHTVLFDLDGTILDTNELIIQSFLHALEGKTPELFTRESIIPHMGRPLIEQLQFFSGLDEVTDLLTLYRSYNLKHHDEL
VKPFPYVKEVLAKLHAYGVKIGVVTSKIRLTTMLGLELCELLPYIQSIVTVDDVTNAKPDPEGIRKALHELGSTAEEAVM
VGDSHYDIEAAKAAGVEAVAVNWSLKGQAYLEQYHPEHMIADMRELYMIFGITEESV
>lcl|NZ_CP026520.1_prot_WP_042232336.1_646 [locus_tag=PC41400_RS03400] [protein=acyltransferase] [protein_id=WP_042232336.1] [location=744329..744910] [gbkey=CDS]
MRKTDRYPVDGPNALWQIYRTVSKWKAVKNFVWIQISRYCPSLPVKNWIYRRMLGMEVGEHTAFALMVMVDVFFPEKIKI
GRNSIIGYNTTILAHEYLTKEYRLGEVRIGSNVMVGANTTILPGVTIGDGAVIGAGSVVHKDVEPYSFVAGNPLRVIRTG
TPDSAEAGTFGASAEISGPVASGNSTNPNSSSS
>lcl|NZ_CP026520.1_prot_WP_042232335.1_647 [locus_tag=PC41400_RS03405] [protein=ATP phosphoribosyltransferase regulatory subunit] [protein_id=WP_042232335.1] [location=745105..746322] [gbkey=CDS]
MSKPKVFEKPTGFRDLLPESVARIREIEFSVLDCMERWGYSQIVTPTLEFYDTVGVASSIEDRKQFKLLDRNGKTLVLRS
DMTAPIGRVVSSLLRDEPLPLRLSYHGNVFRAMEEEAGRDSEFVQTGVELVGDPSAEADAEVVALAIASLQAAGVKTFKF
ALGHVGFLHGLFGETLKDHPEAQEELKSCLLGRDYVGYRTAIRRMGVDEARRGELESILRLRGGKDICEQALSVTSDPTA
QAAIAHLCEVWNVLEAYGVSEHVLIDLTMIGNFSYYTGLTFEGYAADLGFPLCSGGRYDNLLAQFGRPAPATGFALKTNR
ILEAAGDAVRKQQERVLIVYGESRRAEALGKARDLRTLPGKVVVTRHAGTGSVAPWDPAALHKFEQGGYRYEGNVYSDVL
LYLAD
>lcl|NZ_CP026520.1_prot_WP_042232333.1_648 [gene=hisG] [locus_tag=PC41400_RS03410] [protein=ATP phosphoribosyltransferase] [protein_id=WP_042232333.1] [location=746536..747171] [gbkey=CDS]
MQEMLKVAMPKGRIYKQAFKLFRRAGLPMPDDSGDDSRRLIVTFPEANMEFILAKPVDVPTYVEYGVADIGVVGKDVLME
ENRDVYELLDLGIARCRMSVIALPDWQPVLNPRVATKYPQVASQYFRERGQQVEVIKLNGSIELAPLIGLADRIVDMVET
GQTLKENGLVEYEEIFPITSRLIANRVSYRMKNDRIQQLCDALQATISLPE
>lcl|NZ_CP026520.1_prot_WP_042232332.1_649 [gene=hisD] [locus_tag=PC41400_RS03415] [protein=histidinol dehydrogenase] [protein_id=WP_042232332.1] [location=747263..748552] [gbkey=CDS]
MKIVPAEEFELTREVDYGTAEQNDVVKSIIAAVRSEGDAALRRLSKEFDGVDLHELRVDEAEIEAAYDRVDAEFLKALRQ
AAANIRAFHEKQKRVSWMDLQDNGSLLGQIIRPLQRVGLYVPGGKAAYPSSVLMNAIPAVVAGVPEIVMVTPPATAGEEG
INPHILVAAAEAGVREIYRVGGAQAVAALAYGTDTIRAVDKIVGPGNIYVALAKRAVFGVVDIDMIAGPTDIVVLADESA
DPEYIAADLLAQAEHDELSPAILVTPSRELAERVRGEVERQLQALPRKAIAEMSTANKGAIVLVGDLDEGIRVVNAIAPE
HAELLVDDPFAVLPRIENAGAIFLGAYSPEPVGDYFAGPNHVLPTNGTARFSSPLNVDDFIKKSSVISYSKEALLANGEQ
IMTLARNEGLEGHARAIQVRLQKEGNFHG
>lcl|NZ_CP026520.1_prot_WP_272509849.1_650 [gene=hisB] [locus_tag=PC41400_RS03420] [protein=imidazoleglycerol-phosphate dehydratase HisB] [protein_id=WP_272509849.1] [location=748545..749276] [gbkey=CDS]
MAEQVDKNAKQAGLEGGNPGGSVNADAVNADGVDTGAGEGLSVTDSVSRARVSSVARKTNETDIELSFGVDGTGVSELET
DVPFLNHMLDLFAKHGQFDLKVQARGDVDIDDHHTVEDIGICLGQTLREALGDKKGIKRYASVFIPMDEALAQVIIDVSN
RPHFEYRAEYPSQQVGSFTTELVHEFLWKFALEARITLHVIVHYGRNTHHMIEAVFKALGRALDEATSIDPRVKGVPSTK
GVL
>lcl|NZ_CP026520.1_prot_WP_042232330.1_651 [gene=hisH] [locus_tag=PC41400_RS03425] [protein=imidazole glycerol phosphate synthase subunit HisH] [protein_id=WP_042232330.1] [location=749279..749896] [gbkey=CDS]
MIAIIDYGMGNLHSVSKAVERLGCESTVTSDPKILAEADGLILPGVGAFGDAMEQLSSLGLDRLLREQTAQGKPLLGICL
GMQLLFTSSEEYGEHDGLNLLPGQVVRFQGDYKVPHMGWNRLEFKQTSPLLEGLEEGHVYFVHSYHVKPERQEDLLAVTD
YHQPVTAVVGKGSVFGMQFHPEKSGELGMRLLQNFVGLCRSKAKS
>lcl|NZ_CP026520.1_prot_WP_042232327.1_652 [gene=hisA] [locus_tag=PC41400_RS03430] [protein=1-(5-phosphoribosyl)-5-[(5-phosphoribosylamino)methylideneamino]imidazole-4-carboxamide isomerase] [protein_id=WP_042232327.1] [location=749939..750664] [gbkey=CDS]
MSFIIYPAIDIRDGKCVRLVQGDYNQETVYNENPLEVAQAWEAQGGQWIHLVDLDGAKAGRPVNDKLIGEIARTVKVPVQ
VGGGLRTEEDVDRLLSLGVTRVILGTAAIEDRAFVQSLLAKHGENVAIGLDARDGFVATRGWLETSSVKAEDLAVQLAEH
GAKTFIFTDISRDGMMGGPNVEAIVRLAQVSGQTVIASGGVSRTEDLERLAAEAANGVGGAIVGKALYTGSIRLEEALKL
V
>lcl|NZ_CP026520.1_prot_WP_042232324.1_653 [gene=hisF] [locus_tag=PC41400_RS03435] [protein=imidazole glycerol phosphate synthase subunit HisF] [protein_id=WP_042232324.1] [location=750769..751527] [gbkey=CDS]
MLAKRIIPCLDVKDGRVVKGVNFVNLRDAGDPVELASIYDREGADELVFLDISASVEGRATMVEVVKKTAGEITIPFTVG
GGISTTDDMKRLLRAGADKIGINTAAVKNPQLVSDGARKFGSQCIVVAIDAKFNPAWGEWEVYTHGGRTASGIRAVEWAK
RMETAGAGELLLTSMDADGTKDGFDLALTKAVSGFVGIPVIASGGAGNEEHFYEVFTEGRADAGLAATIFHYKELTVQGV
KESLKKKGVEIR
>lcl|NZ_CP026520.1_prot_WP_042232322.1_654 [gene=hisIE] [locus_tag=PC41400_RS03440] [protein=bifunctional phosphoribosyl-AMP cyclohydrolase/phosphoribosyl-ATP diphosphatase HisIE] [protein_id=WP_042232322.1] [location=751524..752201] [gbkey=CDS]
MTGIRLDNIRYDSQGLVPAIVQDAISKDVLMLAYMNNESLEKTAATGTTWFWSRSRQELWNKGATSGHTQRVVSLRYDCD
ADTLLVLVEQTGPACHTGSYTCFGGEIPVAGADSAETGESAAVPASSDRFGILSRLESVIASREAERPEGAYTTYLFEKG
IDKILKKVGEETAEVIIAAKNRDHDELRYETSDLIFHLMVLLRESKLPLDDVMAELERRHMSKSK
>lcl|NZ_CP026520.1_prot_WP_042232320.1_655 [locus_tag=PC41400_RS03445] [protein=ribose-phosphate pyrophosphokinase] [protein_id=WP_042232320.1] [location=752519..753469] [gbkey=CDS]
MDLNNIKIFSGSSNPKLAQSICDDLGVALGKIKLSRFKSGEIYCLYEEGVRNCDVFLVQTFSHPINEHLMELLVMMDAAK
RASARTINLILPYYGYSRQERKAAPREPISAKLVADLVSAAGANRVITIDLHAPAIQGFFDIPVDHLTALDLISDYITKK
NIVDPVIVSPDAGRATTAERLANILDAPFAMMFKKRPAHNEAVVTHVIGDVKGRTPIIIEDLIDTGTTIVNVVEGLKERG
ANDIYVCATHAVFSGPALQRLDHPNIREVVITDSLAIPDDHAERFKVLSVAPLLSHAVRIVSDGGSLSSLFKYGGV
>lcl|NZ_CP026520.1_prot_WP_042232316.1_656 [locus_tag=PC41400_RS03450] [protein=tetratricopeptide repeat protein] [protein_id=WP_042232316.1] [location=753542..755293] [gbkey=CDS]
MEKKKRVSVKPEPKIIPISMDATFFFERAVQSLDRFHYDKALKYFRRAVEYEYDNPVNHCNLAGVLSELGKYEESNDILK
DVLDRIDPSMTECYFYMANNYANMDQYEQAEGALVHYLENDTNGHYLEEAEEMLDFLSYELERPAKLKTIKSREGLYEHD
QARLLLEEGKFAEAVRVLEELVEEHGDFLAARNNLALAYYYTGHFQKSLDAILEVLNIDPGNLHALCNMAIFYQHLGRKE
ELAELMGTLRKTYPFQQEHVFKLATTMGILGDHEKAYQLFRRISRSNDSGLDPCVYHYAAVAACNIGRFQEARVLWNQVD
KLENGSEIARFYLEQLNRRSDDRPLSLSYHYNLPFEEQFRQLEQSPESIPDQIAKDPLVRSSFFWALRHGDVETKLQVLQ
TFVYIGDNEVKEALVEFIGDPKEDDYLKKVAIFVLRTIGVEEPLKAHLDGTVKTVENSVHTTDLPKWEPKWQTVMETAHR
HMHQRYDMIQQHDLQTLWVEYLSRVYPSVPRITKVEGWSAALEYLVAKMHRRSISYHEVASRYHTTVSTVSKNVKLIDEA
CGLKEKMDAILPKLWNVEKKGRD
>lcl|NZ_CP026520.1_prot_WP_042232314.1_657 [gene=trxB] [locus_tag=PC41400_RS03455] [protein=thioredoxin-disulfide reductase] [protein_id=WP_042232314.1] [location=755426..756373] [gbkey=CDS]
MYKTIVVGTGPSGLTAAIYLARANLNPLVIEGPEPGGQLTTTTEVENFPGFPEGIMGPELMANMRKQAERFGAEFKTGWV
NSVELTERPFKLNVEGLGEIETESLIISTGASAKLLGITNERENIGRGVSTCATCDGFFFRGKNIIVVGGGDSAMEEANF
LTRFASEVRLVHRRSELRASKIMQDRARENEKITWSLNSAPLEVVAGDKGVTGLKVLNNETGAEEVLETDGIFVAIGHTP
NTKFLGGQITTDETGYILVKPGTTETNIPGVFACGDVQDLKYRQAITAAGSGCMAALDCEKFLEGNMVHDWSKSL
>lcl|NZ_CP026520.1_prot_WP_042232311.1_658 [locus_tag=PC41400_RS03460] [protein=ROK family glucokinase] [protein_id=WP_042232311.1] [location=756545..757492] [gbkey=CDS]
MTQRIYVGVDLGGTSIKVGLCDEQGKLIQTYEGPTGTEHGPDFVLDRIVEYVHRIVDESPYEWEQVAGIGAGIAGFMDIP
TGFVELSPNLGWRNVAVKKTLEEKLGKTVKIDNDANVAALGEAWGGAGRGIPNLVCYTLGTGVGGGIIINGKIYQGFNGM
AGELGHMSVVPDIEAIACNCGQSGCLETVSSATGIVRMANDAKERGDRTSLALIEKITAKDVFDAAKSGDEVAVRIVNRA
AYYIGKSMAAVAVTVNPQRFIIGGGVSKAGEILFNPIRETFRKLTPERAYTGVEIVPATLGNDAGVVGAAGLNLH
>lcl|NZ_CP026520.1_prot_WP_009676833.1_659 [gene=rapZ] [locus_tag=PC41400_RS03465] [protein=RNase adapter RapZ] [protein_id=WP_009676833.1] [location=757560..758447] [gbkey=CDS]
MEESKAAAKLVIITGMSGAGKTIAVQSLEDLGFFCVDNLPPVLIPKFAELIEQSHGKIGKVALVIDLRGREFFTALQESL
RSIKEDYTLTYEILFLDATDSTLVQRYKESRRRHPLAPNDAPLEGILLERKLLEELKGHASQVIDTSALKPAQLKQKIAS
RFTNMGSGGISINVISFGFKYGIPIDADLIFDVRFLPNPHYVETLRPNTGQDADVYDYVMKWNETQEFLTKLLDMLHFLV
PQYKKEGKSQVVVGIGCTGGKHRSVAIAEYLGKVMGNSETEIVRVNHRDAERDRP
>lcl|NZ_CP026520.1_prot_WP_042232308.1_660 [locus_tag=PC41400_RS03470] [protein=YvcK family protein] [protein_id=WP_042232308.1] [location=758460..759446] [gbkey=CDS]
MSKVYRPADIPRIVVIGGGTGLSVMLRGLKQKPLDITAIVTVADDGGSSGILRSELQIPPPGDIRNVLTALADVEPLLSE
MLQYRFPAGTGLAGHSLGNLILAAMKDITGDFVTGIREMSRVFAVRGRVLPASRQELYLRALMEDGTIVEGESKIPQAGG
RIKRVMLEPADIEPLEEAIEAIREADAILCGPGSLYTSILPNLLVPGIAEEILKSDAVKMFICNVMTQPGETDDYSVGDH
LTAIHNHIGHHLFDYVIVNNGEIPPQVQNKYAEKGAKAVHLDLDEVTKRGYRVIADKLVLFRTYLRHDAEKLSEHIYQIV
ESWMLRKR
>lcl|NZ_CP026520.1_prot_WP_042232305.1_661 [gene=whiA] [locus_tag=PC41400_RS03475] [protein=DNA-binding protein WhiA] [protein_id=WP_042232305.1] [location=759453..760394] [gbkey=CDS]
MSFAAQTKKELTLIEAEDCCEKAELSALIRMNGSVQLTNQRVVLDISTENAAIARRIYSLLKKRYELHTELLVRKKMRLK
KNNVYIVRVPNQVQELLSDLYIVSEGFQFTPGIDKSIVKTNCCRRAYLRGAFLAGGSVNNPEGSSYHLEIASMYEEHCQA
LCQLANTFDLNARCIERKKGFVFYIKEGERIIEFLSIIGAHQALLRFEDVRIMKDMRNSVNRIVNCETANLNKTIGAAVR
QIDNIRLIEREVGLESLPEKLREVAEIRVKHPDLNLKEVGDMLKGSVSKSGVNHRLRKIDELADKIRNGAAIR
>lcl|NZ_CP026520.1_prot_WP_016423762.1_662 [locus_tag=PC41400_RS03480] [protein=HPr family phosphocarrier protein] [protein_id=WP_016423762.1] [location=760664..760933] [gbkey=CDS]
MSRRPVVVKLKTGLHARPAALFVQEANKFSSEIFVEKDEKKVNAKSIMGIMSLAISTGTEIYISAEGSDAEQAVNALVTL
VSKEELENQ
>lcl|NZ_CP026520.1_prot_WP_042232304.1_663 [locus_tag=PC41400_RS03485] [protein=copper amine oxidase N-terminal domain-containing protein] [protein_id=WP_042232304.1] [location=761085..762842] [gbkey=CDS]
MIQWKKEVSLLMALVLFAAFLMPIQARAISSSSSLIVLYLDNKQAFVNEDQVELESPATIIKDKTFVPAKFIGDSLGFPV
TWDNAARTITMQPPGYNIVLSSDLKKAYINGEEVPFDSVAAIVKGKLLVKLTWVADYMGAKYTYYHDSRKVEIIYVPKPE
GIYDNGTNNSNPVAKFTTHKEKYRLGETVKYIDLCYDPDAEGIAKYEWTGKQDAFFKPGSYPVSLKVYDRHGHVSKVFTK
NVIVENIPLVSEAEYPVYHNEIGSFYKLSWDTLTSHFYNLPMLPKEATDVTGRKLLVSDSPEEFTEPGILYQDMINGKGR
LYADHINGTKNKVTFAIMATNTTDKPVTVKTTNKGEVWPSVYANLIGSEASVDFLLGNTYNEKLTIPPHDTLVYRQLPDF
YPGQGVNLFYDLETSGEVQFSFLAADELSNTTLSESKLLGFTGHVRGSFDVSEVNWNIDASSFTTASKLVIGDGKTDPFV
DGYDVMRKENVKNGGNYGVVYKIHADKPRKMAIMLLARGGVFKGPIKINGEIQSIPYSGVITAFDGMQILHRTTGTEKSL
DIEFTPPAGSAFPVDLIFYPLADLD
>lcl|NZ_CP026520.1_prot_WP_009676841.1_664 [gene=clpP] [locus_tag=PC41400_RS03490] [protein=ATP-dependent Clp endopeptidase proteolytic subunit ClpP] [protein_id=WP_009676841.1] [location=complement(762925..763527)] [gbkey=CDS]
MSFIPMVVEPDARGERAYDIYSRLLKDRIIFLGTPVNDQVANAIIAQLLFLDAQDPDKDIHLYINSPGGSITAGMAIYDT
MQFIKSDVSTICVGMAASMGAFLLAAGAKGKRYALPNSEVMIHQPLGGAEGQATDIEIRARRILKMRENLNKILAERTGQ
PLERIDKDTDRDYFMSAAEASAYGLIDKVIEKTTEGGKEA
>lcl|NZ_CP026520.1_prot_WP_042232301.1_665 [locus_tag=PC41400_RS03495] [protein=DedA family protein] [protein_id=WP_042232301.1] [location=763882..764532] [gbkey=CDS]
MEWISDLFAQYGYLVLFLGLFAESVALPFPGELAMMISGHMVHVNTFNMAGVMGVSFAGAFAGTIFTYVLGRKLGTPFFD
KYGKYVLLKPERLLKIKKWFDKYGDRIILVSYFVPGLRHFTGYVSGILKIRLKTFLIFNGLSGVLWVFTYVMVGFLFGTK
IQQLLHVISQYSTIAIIAAAAALTGFLVVKRNKEAVLGWLRARYAGIQKMFMGSAE
>lcl|NZ_CP026520.1_prot_WP_042232298.1_666 [locus_tag=PC41400_RS03505] [protein=sugar-binding domain-containing protein] [protein_id=WP_042232298.1] [location=764920..765948] [gbkey=CDS]
MRDILDVQKQLLPDLLEVLNKRYTVLQHIFLLGSVGRRTLAGALDMTERVLRSEVEFLKGQGLIETETSGMRISESGREL
LEKLQPLIKEMLGLNELEKRIRDHFRLEQVVVVPGDSDSSLFAKKELGRAAALMLRKHVTKHDVIAVTGGSTMAEVAEHL
STTSALKGSLFVPARGGLGESVELQANMIASKMAKKTGGQYRLLHVPDHLREEAYQSLIQEPNIQEMMDVVRSSRIVIHG
IGDAIVMAQRRKVDSETIDSLRGEGALAEAFGYYFNPEGKVVHKMPTVGLRLEDIQHCEVVIGVAGGKSKGAAVASVLKY
GQEDILVTDEAAAVEMLNHMLS
>lcl|NZ_CP026520.1_prot_WP_016423763.1_667 [gene=gap] [locus_tag=PC41400_RS03510] [protein=type I glyceraldehyde-3-phosphate dehydrogenase] [protein_id=WP_016423763.1] [location=766000..767004] [gbkey=CDS]
MVKVGINGFGRIGRNVFRAALNNAEVEIVAINDLTDVSTLAHLLKYDTTHGRLNATVEAGEGELKVNGKSIKVFAERDPG
NLPWAEYGVEIVVESTGIFTDKAKAESHLKGGAKKVIISAPATNEDITVVMGVNEEKYDPSAHTVISNASCTTNCLAPFA
KVLDEKFGIVKGMMTTVHSYTNDQQVLDLPHKDLRRARAAAENIIPSTTGAAKAVSLVLPQLKGKLNGMAMRVPTPNVSV
TDLVVELNANVTVDDVNNALKEAADGPLKGILNYSDEPLVSSDYNGDPASSTIDSLSTMVVGDNMVKVVSWYDNEWGYSN
RVVDLAAFIAKKGL
>lcl|NZ_CP026520.1_prot_WP_042232295.1_668 [locus_tag=PC41400_RS03515] [protein=phosphoglycerate kinase] [protein_id=WP_042232295.1] [location=767409..768590] [gbkey=CDS]
MNKKSVRDVEVNGKRVFVRVDFNVPLENGQITDDTRIRETLPTIKFLIEKGAKVILASHLGRPKGQVNEDLRLTPVAARL
SELLGKPVAKADEAVGEAVQAKVNELKDGDVLLLENVRFYPGEEKNDPELAKQFAALADLYVNDAFGAAHRAHASTEGIA
HQLPAVSGLLMEKELDVLGKALNNPERPFTAIVGGSKVKDKIDVINKMLEIADNILIGGGLSYTFFKAQGHEIGQSLVDN
EKLDLALGFIEKAKKLGKNFLLPVDIVISDDFSANANTNIVGIDSIPADWEGIDIGPKTRELYADVIKNSKLVVWNGPMG
VFEIEPFSHGTRAVAQACADTSAYTVIGGGDSAAAAEKFHLADKMDHISTGGGASLEFMEGKVLPGVVALNDK
>lcl|NZ_CP026520.1_prot_WP_042232293.1_669 [gene=tpiA] [locus_tag=PC41400_RS03520] [protein=triose-phosphate isomerase] [protein_id=WP_042232293.1] [location=768618..769370] [gbkey=CDS]
MRKPVIAGNWKMFKTVSEATSFVNDIKGKAEVAGVESVICAPYTNLPSLVEAVKGTSIGIGAQNFHWEENGAYTGEISGA
MLKDLGVTYVIIGHSERRAYFNETDETVNKKVLASFNYGLTPIVCVGEKLEEREAGQTKEVCRVQTEAAFKGLSADQAKQ
VIVAYEPIWAIGTGKSSTAEDANEVIAYIRELIASLYDSSVADAVRIQYGGSVKPGNIAEYMAQPDIDGALVGGASLEPA
SYIQLVEGAK
>lcl|NZ_CP026520.1_prot_WP_042232291.1_670 [gene=gpmI] [locus_tag=PC41400_RS03525] [protein=2,3-bisphosphoglycerate-independent phosphoglycerate mutase] [protein_id=WP_042232291.1] [location=769372..770913] [gbkey=CDS]
MATKPVALIILDGFGLRGEVHGNAVAQAKKPNYDRYWNQFPHTTLTACGEAVGLPEGQMGNSEVGHLNIGAGRVVYQDLT
RISKSIRDGEFFDNNTLIGAVRHAKQNNKKLHLYGLLSDGGVHSHIAHLFALLELAKKEDLQEVYIHAFLDGRDVAPDSA
KNYMEQLQAKIAEIGVGKVATVQGRYYAMDRDKRWERTEKSYRAMVYGDGPHYTDPVKAITESYEKSVFDEFVMPTVITD
ENDKPVGLVESGDAVIFFNFRPDRAIQLSQVFTNKDFRGFDRGELFPKDLYYVCLTLFSETVEGYVAYKPKDLDNTLGEV
LAQNNMRQLRIAETEKYPHVTFFFSGGRDVELPGETRILIPSPKVATYDLKPEMSAYEVAAAAVKEVNEDKQDVIILNFA
NPDMVGHSGMLEPTIKAVEATDECLGQVVEAILAKGGVVCITADHGNADLVLDDAERPFTAHTTNPVPFIVTSNDVTLRE
GGVLADIAPTLLHLLKLAQPPEMTGLSIIEEKK
>lcl|NZ_CP026520.1_prot_WP_009674533.1_671 [gene=eno] [locus_tag=PC41400_RS03530] [protein=phosphopyruvate hydratase] [protein_id=WP_009674533.1] [location=771074..772366] [gbkey=CDS]
MTIISDVYAREVLDSRGNPTVEVEVYLESGVVGRAIVPSGASTGAHEAVELRDGDKSRYLGKGVLKAVDNVNSIIAPEII
GLNSLDQVSIDKKMIELDGTHNKGKLGANAILAVSMAVARAAAEAMELPLYVYLGGFNAKTLPVPMMNIINGGEHADNNV
DVQEFMVLPVGAPTFKEALRMGAEIFHNLKSVLKDKGLNTAVGDEGGFAPNLSSNEEAIQTIITAIERAGYKPGVDVFLG
MDVASTEFFKNGKYELEGEGKSFTPAEFVDLLASWVDKYPILTIEDGCSEDDWEGWKLLTEKLGGKVQLVGDDLFVTNTE
RLSKGINENIGNSILIKVNQIGTLTETFDAIEMAKRAGYTAVISHRSGESEDSTIADISVATNAGQIKTGAPSRTDRVAK
YNQLLRIEDQLEGIAQYAGKEAFYNLKKFK
>lcl|NZ_CP026520.1_prot_WP_042232290.1_672 [locus_tag=PC41400_RS03535] [protein=M50 family metallopeptidase] [protein_id=WP_042232290.1] [location=772425..773150] [gbkey=CDS]
MSAWVKTILLLIGSALLTQFIPFSAYFRNLDTMVHEFGHAAVTMILSGKVMAIVLNADHSGVTYSSVSGSWKLLPISIAG
YVTASLFAVFLFWANARGKQRLGLQVTTVVAVLSLVLFVRNEFGVGWLIGFIALNIIVLAFTSGFIRNAYYLLVAFLCLE
ESVYGPLTLALLAAGNPNKAGDATNLASITPIPALIWGVVFLVFSLWCAKAAIQYFLGGDANRRKKARRSERAYPMGPGE
R
>lcl|NZ_CP026520.1_prot_WP_009674538.1_673 [gene=secG] [locus_tag=PC41400_RS03540] [protein=preprotein translocase subunit SecG] [protein_id=WP_009674538.1] [location=773335..773571] [gbkey=CDS]
MVILLKVLLVIASLGLIVTVLLQKGKSAGLSGAISGGAEHLFGKTKARGMDLVLARITVVLAAIFIVLTVIVAYVVKY
>lcl|NZ_CP026520.1_prot_WP_241688877.1_674 [gene=rnr] [locus_tag=PC41400_RS03545] [protein=ribonuclease R] [protein_id=WP_241688877.1] [location=774083..777007] [gbkey=CDS]
MVTEQDILNFMHENAYKPMTYQELEKHFGIESASEFKDFVKLLNDLEQSGHVLRTRNDRYGVPERMNLIRGRLQAHPKGF
GFLIPDDRDHPDVYIHANDMNTAMNNDIILVRVTSKGAAGGKLEGEVDRVVTRANTRIVGTFQNHESYGFVVPDDKRVQR
DIFIPKNAFLGAADGQKVVVNIVQYPEGRAAAQGEVIEILGHKDDPGVDILSIIRKFQLPEAFSEEVIGEAESVPDSITE
EEIVSQKRRDLRGKRIVTIDGEDAKDLDDAVNVERLPNGNYLLGVHIADVSYYVTENSALDREAYNRGCSVYLVDRVIPM
LPHRLSNGICSLNPQVDRLTMSCEMEFDAELNVVRHDVFTSVIKTSERMTYNNVRKLLTEEEPDADLLERYGYLVDDFKL
MEELAMKLRNRRIQRGAIDFDFQESKIIVDKDGTPVDIVKRERSIAEMIIEEFMLAANETVAEHFHWLKVPFLYRIHEDP
DAEKLAHFMEFITNFGYVVRGRGNTVHPRALQSLLEEIKGTKEETVISTVMLRSMKQARYDSNSIGHFGLAAEFYSHFTS
PIRRYPDLIIHRVIREVLENGGALTDKRSEYLRGRMDDIAEQSSQRERVAVDAERETDALKKAEFMLDKVGEEFEGIISS
VTSFGIFVELENTVEGLIRLSDMTDDYYHYHEMQHALIGERTSKIYRIGDEVKVRVGRVNMDEKTIDFEMVDMKPRSNKF
SMLDRSKSGGRGGQKKRVAGGRPVKDGRGKDRGRVRDVIVKPGSPGGAAGAADAAGASEGGGTKRRRRKKKGASGEAPLG
ITPAAAPGGTPSEGGSLSQEAPGAGRGRKRRKGGNPAAKPDTGRENGLVTESGENGGGRRRKRKGGGAEGEAAVTPAAGT
DGGGKRRKGAAPARGVSLRGESGEGGGQGGRGRRGRDAGEGAGRESGGARKESTYSGNWASSVNALTKTVRGDRHTPDGR
DPFEPGGRRGRPRR
>lcl|NZ_CP026520.1_prot_WP_009674535.1_675 [gene=smpB] [locus_tag=PC41400_RS03550] [protein=SsrA-binding protein SmpB] [protein_id=WP_009674535.1] [location=777399..777878] [gbkey=CDS]
MTKKADGKVLAQNKKASHDYFIEDTYECGLVLTGTEIKSLRNGRANLSDSFATIRNAEAFVHNMHISPFEQGNRNNPTDP
TRARKLLLKKAEINKLIGASKQEGYTLVPLKIYIRNGYAKLLIGLGKGKKQYDKRDTAAKRDAQRDIQRALREKQKVAR
>lcl|NZ_CP026520.1_prot_WP_042232286.1_676 [locus_tag=PC41400_RS03560] [protein=AraC family transcriptional regulator] [protein_id=WP_042232286.1] [location=778765..779580] [gbkey=CDS]
MIRTFEPDNVLHKLVDYIWIVDFNFLADGNRGDIIMPLGHINIIFNYGSAYRLVGEKEEMVIPDVAVIGQMKEAKHVQYG
QHLYQIGISLTPLGFIKLFHEPGLELTERIVEAIDVDPDLDELYLMMMISKDVEQCTKAINQFLMHKLALNEKDTGRIEE
MLTYVEREYESLNIVSMANFFGVSVSSLERFFKKYIGLTPKTYGNIVKFRKHVEDEGLRRDMQYHYYDQSHLIKTSKKFA
GKTIKELEELPDELTLGYLWHGQQQASEHRC
>lcl|NZ_CP026520.1_prot_WP_053228758.1_677 [locus_tag=PC41400_RS03565] [protein=hypothetical protein] [protein_id=WP_053228758.1] [location=779651..780040] [gbkey=CDS]
MQDMMIIASLLVFLNVTLLTILVPGGPIENRDFSKLTGVVFWGFNLFLISLGVVSFIACYLLLISHSNAILITQMIAVLY
FIVYTIDLAGMFPKSPTKMSKPLMLFEIINTTMAVFLFLFVTAIGHAGS
>lcl|NZ_CP026520.1_prot_WP_241688781.1_678 [locus_tag=PC41400_RS03570] [protein=zinc ribbon domain-containing protein] [protein_id=WP_241688781.1] [location=complement(780259..780819)] [gbkey=CDS]
MDNNKRQSGAYKAKEVYLLSGLIVCGKCPKKIGTPYSMMGNIKFSGSNKLKYVTYRCGNRDRTKEWNNPEIRREYIESYI
ISQLQEKTFNEEAIPLLAKQLNDYHNSKQSNVKGKRERLAKSLEGIERQITNIVNAIASGTFNIALSSKIWELKQQKLQL
GQKLLEAKGVTEKAAITEDTLRQLLF
>lcl|NZ_CP026520.1_prot_WP_241688878.1_679 [locus_tag=PC41400_RS03575] [protein=amino acid adenylation domain-containing protein] [protein_id=WP_241688878.1] [location=781683..784886] [gbkey=CDS]
MYPSTKVSILSVTVKMKGKVYPDLLEESIQWVIQENDAFRVKIDSKDGEPRQCVEPYKKKEIRMIDFSDYTDSSHAEEWL
KSHNRMPGEQFASELYQFVILKISNEELWYSLQIHHIVCDGIAINLTINQIDRNYAALLKGTSSSEHVTNAYLNCILDEL
DYEKSERYSKDQAFWMEKYRNLPNITGIKNYNPLLTSIAAERKSFTLSDELYRNLKLFCQNHQVSMFTFFMAALYIYIYK
VTNESDVVIGALYGNRTSKKVKETIGMFASTVAIRLDVEPDLKLNAFLQKVSKEQSVMLRHQKYPYTKLIQDVRRMHPSK
DIQRLFGIAVEYQPYNILKFEHSDMEIEADFCGDEVNDFIIHIKDCLDTQELVLFVDYRSQLYDENEIQTMFQQIVTVIE
QMIRCPNDKIADVSLADEDVKNTLLTLFTETRVAYPQEKTIHRLFEEQAAKTPEQIALICGGERVTYEELNRKADQLADV
LRMNGVAREEIVALLADRSVLLVVGMLAVNKAGAAYLPLDPDFPEERIQGMLEEGGCRFVITESAYAGKGTTGRKTVLLD
REEEWPLLSSLLPAVEGACPSDLAYVIYTSGSTGKPKGVMVEHRSVHNFLLGMLQETPIDRVRTVLSLTTVTFDPFVLET
LLPLMLGCTVVMATRGEQTDPDQLADLCRKHGVELLQATPARLQMLLGSQKMAEAMAGLSYVLFGGEVLTKKLLYRLRQA
TQARLFNLYGPTEATVWATVKDVTDDTRLTIGRPIVNTRIYIVDVRGRLQPVGVPGELCIAGMGVARGYLGLQELTAERF
VPEPVEDKADRVGRMYRTGDRARYMVDGTIEFLGRMDDQLKIRGYRIEPGEIAAVLSGHPLIHAVAVVDGQVEQGGAYLA
AYYESTEPISAAQLREHVGRQLPTYMIPSYFVRLDSLPLTSNGKIDKRVLPRPQALTTSASSDSLPRNEGERLLSVLWKE
VLGLTEVGRHDNFFDLGVHSMHAVQLVERIRKEGYDEIKLTDFFMHPTIASLADHLFGHRLTKKWASAAEERAEVRKAMM
RKQRESRGAMEVKKLGNTTGGEAFEPR
>lcl|NZ_CP026520.1_prot_WP_129112449.1_680 [locus_tag=PC41400_RS03580] [protein=beta-ketoacyl synthase N-terminal-like domain-containing protein] [protein_id=WP_129112449.1] [location=784873..787590] [gbkey=CDS]
MNQDKSLWEQGIAVIGMSVKFPGAGNPEAYWNLLSRGVHAVRMFSDEELREAGIPENVFNHPQYVKAKGVLEDPELFDAE
FFGMSPRETELTDPQQRLFLECGYEALEDAAYVPDTYAGRIGVFGGCTISTYMLSQIAKQGPHSNLEELTALMLGNDKDF
LTSRLSYKLNLRGPGMTVQTACSTSLVAVHMACQSLMSGESDIALAGGSSVTFPNHAGYRYQQGLIFSPDGVCRAFDHKA
QGTVFGNGVGVVVLKRLEKAAEDGDFVYAVIRGSAVNNDGGLKAGFTAPSPEGQAEVIADAIAAAGVHPDTISYVEAHGT
GTAVGDPIEVVGLTMAYRQWTDRSGFCAIGSVKTNIGHLDCAAGVAGLIKTVLALRHRQIPPSLHFHEQNPQIDFVNSPF
FVNTKLTDWKVESYPRRAGVSSFGIGATNAHVVLEEAPNREEVAYVRPWHALLLSAKTETALEAACERMSAYLSENPLTP
LEEIAYTTQVGRKAYPLRRAVVCQNAEEAGRLLSRRTEGGTVRHSDEFPAMAWIFMPWREDRLPGLLELCRTEPVFRREL
QEALARAEAVSGLALNALTSVLCSDDTHYTQMPQLTSRMRVVLSVAASYSLAAVFGQLGLQPALVGGEKEGGYAAACIAG
LYSIEDAFRLAIARAIFEEGGEEEAYTRTVGDIFHAAARIPYLNGDQDDFRSDPDQPSAERWLHNAHRAALQDSHMRLVS
IGAPLGDESEGTLSCFDAYSWTDTLADLWEGGADLDWQARTSLEKRRRVPLPTYPFERRRYVIRGPEGHNVDWEAERKAA
VHETQVEPADSGQFEDGNPRRTLERNLILFYQEVLGQPSIDIHVPLYQLGADSLSKMQVISRIQETYPILLNVERMYGAQ
TIAELADSIELSFIELLEQDNLIRG
>lcl|NZ_CP026520.1_prot_WP_042232284.1_681 [locus_tag=PC41400_RS03585] [protein=non-ribosomal peptide synthetase] [protein_id=WP_042232284.1] [location=787598..792322] [gbkey=CDS]
MTNVDSMEARRSQLSEEKRALLGQLLQSAVVPVNTKQGISVRQKQEKSPLSFAQQRLWFLSSLDPASTAYNLVYAIKLKG
ILQDGALERSLQAVVQRHEMLRTVFGDEEGSAYQIILDTMNVPLERKDLRDLPGGKQEEAIQEQISLINRVFDLRCGPLL
RTALLYLAEEEHVLLLAFHHIVTDGWSHHRFSRELMEFYNSFVCGNAATLPPLRLQYADFANWQREWVQGAAVQEQLSFW
KRKLAGAPALLQLPTDYPRPLTQQYIGRKIHFTVPQQLKERLCRLAAEEEATSFMLLMAALQSLLYRYTEEGDICVGTPL
SGRNRKDVESLIGFFVNSLVIRSEVNAGLPFRQLLRHVRKNALDAYANQDVPFEMLVEETAGQRSLSYSPLFQVMFSMQN
APAVSFELTGLTAEQLDVETETSKFDLSLEMFETESGWKGALEYNRSLFEEATIQRLTGHFTMLLSSIADSPDAQVGALP
VLSGEERELLNGWNRTSHDYPLQTTLHQLVQAQAVGQLERPAIVNASHHITYGQLLGNAGHLAERLREFGVGPGMITAIC
LERTPEMVVAQLAVLMTGGAYLPIDPILPKERITYILNDSLASVLLTTEQLRPFIAGTIVAQILYLDNETDPSGDFPLCD
FTDAHVSSRKQFYEETAYVIYTSGSTGQPKGTLISHESIINFLHWYRDATALSSSDRVCFASGVGFDLSVAEIWGALVSG
ASLWLPSEETRLQPELLRDWMLDNGITFAFLPTPLAEQLVSVRWPAETPLRRLFTGGDKWTAKLPCGLPFEVYDLYGPTE
CTIASTFRMIEPDGEAYQPPHIGRPLANMKIYITDRELRLMPIGVPGEICIGGQGVGGGYLRREELTREKFIPDPFSACE
GARIYRSGDIGRYLPDGRIQFLGRADDQVKVRGFRIELAEIQEALATHSSVKEVVVLVREETPGDKRITAYMVGKSGLTL
DVESLRQSLREQLPSYMIPSAFVILDRLPVTPSGKIDRQALPMPDTDLSDDRFEEPRNFVEELLCGIWKELLVLEKVGVH
QNYFEIGGDSIKTMMLISRARKLGLELTTKQVFQYQTIAELSQAIGVDSRRMEEPPKETVVGLKFDFPLVGLNREDFASI
EGVTREEIEDVYPLTPLQDYMLDTMRRRPEPAQFFVNMVMRFKDKLEPELMTEAWQRVANHFALTKTSVVYEGLAEPVQL
TRSSIRVPVQYLDWRGWDAERQQEELKRYQERNLLRSDLTYLRRSTTYDVMFATLGENNHQLVMSCSYLMMDGWSHFIVL
IQVIKCYYSLLEEKPYELPPAGHYGKYAAWLRSRALEEPRRYWSAELAGFCRATPLVEHAPFNGQPRGKGFAKKSVQIKP
AVPHQVRELALRSRVTVNVLFQLAWSLLLARYTDEREVLFGVMSNGRQPEYEGAEDIVGPTINTLPMRVRVDEDETAFQL
LQKIQEKQLLLTQWDYTPLRKIREWIGMSEEKALFESYMIFQNLPSYFEVSGGKDWIRDVPLENEYEKALAIFDSGTPLR
VDVCVIPEGYEIYMTYLKACFRDEAVSRMLEDLQKMFLGICEASEQPVRTWLRS
>lcl|NZ_CP026520.1_prot_WP_042232282.1_682 [locus_tag=PC41400_RS03590] [protein=hypothetical protein] [protein_id=WP_042232282.1] [location=complement(792752..793270)] [gbkey=CDS]
MQKEELDKLFKKSKLPFRYSVKDAEKILTDRYNTIEIDKEIVNDLKKDPLVNYENIIKCYRVNDSKIHSALFNDNERSMH
AETRQRNANEPISNSEITESMLLWRNIQEGKFLQLVLESADREYISSFTLQGLSEKMLDELIILMGIPPEEGYIGNPAYE
SYLNVLHKNELI
>lcl|NZ_CP026520.1_prot_WP_042232280.1_683 [locus_tag=PC41400_RS03595] [protein=hypothetical protein] [protein_id=WP_042232280.1] [location=793767..793964] [gbkey=CDS]
MFANREQLQEVLKKANQHAREQAKESGASIYYIKNNKRVREDAEGNKFEIIFDATGKRHEFEYHE
>lcl|NZ_CP026520.1_prot_WP_042232278.1_684 [locus_tag=PC41400_RS03605] [protein=DUF262 domain-containing protein] [protein_id=WP_042232278.1] [location=794642..796387] [gbkey=CDS]
MYRVTSFKIRQLYQLFQDGKLFVNRRYQRNLVWPEKDKKYLVDSVFRMYPIPAIMLNMSEGEVYEVIDGIQRLHSLFEYI
EGKFGFERGDDFIFFKDYSDSCDSDSSRKYISEAESQIFLNAQVPIALCEYKDDSDVDNIFLRINSYGISLNPQEIRQAG
KKTAFSSLIKKIGLILRNDMISDQLFSSESIGFIEEIKRNTLGYETNIEKSFWCNHQLFSLNDIKRSEDEEIISDVILSI
VLEKQYPAGRAELNNYYGVGEIDKTIKIDQEIKNYGADKLVQEVVGVFNHIAYMCREELEPQSISLRELLNGKSKEESLE
AFYSICLVFHELLYKNGKKPSDYKEICIALKNLTNRLSYENDLSKSRELNVNQCKGLVETYFRPTDQTPFYTTSTELEQL
IRNSTLEKPEYDFKQGMYHLGPNERKFNKGMYDTIYCTLASLANLGKNRTGYLFVGIADKEKDTAQIEKLDRISVSRINN
FGVVGLEREALLYGSSLDNYVSMIIQGIRNSDLPERLKTQVNSTIEPIKFKEKTVLRITVRCGENPVWYQGKLYIRDGFS
NYRGVSSSESEYIDSVYSLFK
>lcl|NZ_CP026520.1_prot_WP_042232275.1_685 [gene=rbsK] [locus_tag=PC41400_RS03610] [protein=ribokinase] [protein_id=WP_042232275.1] [location=complement(796967..797896)] [gbkey=CDS]
MREPNIVVIGSLNMDIAVEAERSPSIGETVMGTDVHFIPGGKGANQAVATSRLGAHTTMIGTLGSDDFGAELRNSLVNSG
VDVLGIKTAEGIPTGIASILLAQKDNQIVVVPGANAHCLPEDVYRHEALLAQADVVLLQLEIPLETVVAAAEVAKRHGKT
VILNPGPARDLPEELLKHTDYITPNRLELALLTGMDTSGEQWKDAAKRLLEMGPRHVITTLGSQGSAYIKPDMHITTVPA
YRVPVIDTTGAGDSFNGGLAYALAVGDDLDAAVRFAAKVSALAVTKLGAQAGMPSLEEVETFCWSNVQG
>lcl|NZ_CP026520.1_prot_WP_042232274.1_686 [locus_tag=PC41400_RS03615] [protein=nucleoside hydrolase] [protein_id=WP_042232274.1] [location=complement(798014..798943)] [gbkey=CDS]
MTRRKIIMDCDPGHDDAIAIILAAVQPKLEILAITTVSGNAEIEKTTQNALKVCDLISLHDVVVSKGASEPLVRIRETAP
GIHGDSGLDGPKLPEPSRSWSEEHGVDTIIRLVKESPEPVTLLPTGPLTNIALALTKAPEIKNNIEEIVLMGGGTFGNWT
PTAEFNIWADPEAAKKVFDSGIPTVVMGLDITHQALATKEVIEEVNKIDNHVAKIVGELLVFFASTYKEMFDFDGAPVHD
VLTVAYCVAPELFKTKDVNIVVETKGEYTAGTTLVDLHGVTGRKVNAKFGLELDVDGFWKLMIDALKKY
>lcl|NZ_CP026520.1_prot_WP_042232271.1_687 [locus_tag=PC41400_RS03620] [protein=nucleoside transporter C-terminal domain-containing protein] [protein_id=WP_042232271.1] [location=complement(798966..800147)] [gbkey=CDS]
MNVIFLITGLLLVFMIGFLVSNNRQKIKYKRVLIMIAVQILLVYFMTNTSIGIIVITKVGLLFEKLMEIANSGIQFVFGG
LANEGQSPFFLGVLMPLVFMSVLIGILNYIRVLPFIIKYLGLFLSKITGMGKLESYFAISTSALGQPEVFLTIIKQIPLL
SKERLYTICTSAMSAISMAMVGAYANMLEPKFVVTAVVLNIFSALIVANIINPYDLDENEDLVEIDDKERLPFFRMVGES
VMDGFKIAIIVAAMLLGFISLMQLVNVIFQSAFQVSFDTVLGYIFAPIAFLMGVPWTEAVQAGGIMATKLITNEFVAMLN
FREITNQLSSKTVAIVSVYLVSFANFGTLGIISGSIKSISDKQGDHVAKFAFKLLLGSTLASVISGTMMGLVM
>lcl|NZ_CP026520.1_prot_WP_042232268.1_688 [locus_tag=PC41400_RS03625] [protein=winged helix-turn-helix transcriptional regulator] [protein_id=WP_042232268.1] [location=800473..801567] [gbkey=CDS]
MDKEKQILDYIKMNPYISQQELSSKVGLSRPTVANYIANLTRRGEIKGRAYILREESSIICIGGANIDRIARSNQKVNFH
SSNPVKINESCGGAARNVAENLSRLRCNTSIMTCVGDDKEGNWILNEIKRQGVDVSQVWVLPTERTGIFTTLLDITGESV
VSMDDMHIYEQMTTSMFEEKWSYIVSSRAVFLDTNIPKECISYLIKRCADEQIPVYMDPTSAAKTRKLPLRLDGVELLLP
NRVEAEILADMKIDSIEDCQLACEKIRERGVKKVIITLGDQGAYYSSSEECGHLPSFLTDIVDTTGVGDAFSSCAIYGTM
NNESLYSACQSGLAGAALALQTEQSICTSLRPEKIYEVIQEYSR
>lcl|NZ_CP026520.1_prot_WP_156972847.1_689 [locus_tag=PC41400_RS29640] [protein=hypothetical protein] [protein_id=WP_156972847.1] [location=complement(802026..802163)] [gbkey=CDS]
MTIGASEAAGTPEATGATGAIRTIETTRTIETIEAPETIETGEEA
>lcl|NZ_CP026520.1_prot_WP_042232266.1_690 [locus_tag=PC41400_RS03630] [protein=putative thiazole-containing bacteriocin maturation protein] [protein_id=WP_042232266.1] [location=802493..804451] [gbkey=CDS]
MANLNPSMRLKVKRDTFFLPDLSGVYFRNNVSSFRMEGEAIDQWIETLIPIFNGEHRLEDLTDGLPDQHRNQVYEIAEVL
YRNGFVRDVSQDTPHQLPEEVLTKYASQIEFLDHFGDSGAYRFQCYRQTKVLLVGSGPFLISAISALLESGLSKFYVLVS
GTDSTDRQRMAELVAHARITDSEVAVVEVALPKEEGSCWREAVRPFDSIVFVSREDDARELRVLHAACREEQKLFLSAVC
LQQVGMAGPLVHPDFEGCWESAWRRLHQSAISKDPQLHTFSSTAAAMLANVSVFELFKKITGADDSEQNNRFFLLNLETL
EGRWHSFLPHPLVTELSGPKWIHDFDLQLERRSNKSENGLIPYFSRLTSAESGIFHVWEEGDLTQLPLSQCRVQAIDPLS
EEPAGLLPDIICADLTHEEARREAGLAGIEAYVSRIADLLVPMLPPHQGEGADRIEPQEFIGIGTGEAVAEGVCRGLQRC
LEDVLGKRQASRLPVVSRVQLRTVEDKHCLFYLEALTTMKGAPLIALGEEVSGFPVVWVRTSDRWYCSVGLNETMALRKA
LQQALLGAQIQPDGLKSQALGFSSVLLKEQAPLNLEIPACEAASQPAVLQSALQVLNRNRKKLLVFDLAAEPFLKEEVAG
VFGVVLREEGSR
>lcl|NZ_CP026520.1_prot_WP_042232264.1_691 [locus_tag=PC41400_RS03635] [protein=TOMM precursor leader peptide-binding protein] [protein_id=WP_042232264.1] [location=804448..806391] [gbkey=CDS]
MSAAVVIVGEGVLADCVCGELSTQFEVVRQNDFEAGVPEAADLVLVLNDAWNPYIHQKAEELLQPVGIPWLRGFVSFGEG
VIGPLVRPGTQGCSQCADMRRLMAGRERKEMRGLQQRLMTRGGVDRDAWASRTGLLQVGHLLAVEARKVLQGQTVHSAER
MFLINLKTLEGTLHFVLPDPLCPVCSQLLDDSSAAAHISLQPSPKISEDSYRCRGMDELNKVLTQDYLDYRTGFLNGKMV
DLKSPFADVSVNLPLLAEDVGTAGRTHSYAVSELTAILEGLERYCGLAPRSKRTVIRDSYRNVKDQALDPVKVGEHTKEQ
YAQPGAPYTPFNPDHPIDWVWGYSFLQERPILVPEQLAYYSMGCGHSFVFETSNGCALGGSLEEAIFYGILEVVERDSFL
MTWYAQLPLPRLDLYSANDQELQLMIDRLRAVAGFDVHFFNSTMENGIPSVWGVAKNRKQTGVNLICAAGAHPDPVRAVK
SAVHELAGMVQSFDEKLDTNRETYVRMFHDSSLVQQMEDHSLLYGLPQAQERLQFLLDDNRPLRTFEEEFKREARHADLT
DDLKDILKAFHRLNLDVIVVDVTAPEIMRNGLHCVKVLIPGMLPMTFGHRFTRLAGLERVLRVPVELGYAEKPLRLEDLN
PHPHPFP
>lcl|NZ_CP026520.1_prot_WP_042232262.1_692 [locus_tag=PC41400_RS03640] [protein=SagB family peptide dehydrogenase] [protein_id=WP_042232262.1] [location=806447..808027] [gbkey=CDS]
MSLETFLDNLHYDIDKTKPPDLAVDWEDGPLAYKLYRGLPVVPLSPEIPLTLAGGEAPVKPDLRRVGHFLWYVFGLTQLS
QSVLASNSMVQSMGLMQMYRRFAPSGGALYPNEVYVYLKIEDLPTGVYHYDVAHHRLVLLREGNFDSYLAGAMGNRYDVL
ACFGTVFVSTLFWKNFFKYNNFSYRLQGLDAGALIGQLLEVAKRFGYASGVYFRFLDRAVNYLLGISEQEESVYAVIPLS
VEPEISWFANSNEGERSVSATELCRELPTVQHDHYVRSRRVREFPMLIKMNEASLLESLRSIPRTGEKKIAHCGGHVVVL
PYVKRLLYDLASVSRRRYSPDSDFRMSKVSQEQLALLLREATIPSRDWNDLNGVRGNPVSRVSLYVCLYNVEGIPEGAYH
YDSAAHALRLVRHGDYRLQLQQGMPSDNVNLFQVPLCFHIAGDKDHLATALGYRGYRIQQMEAGILVQRLLLAASAIGLG
GHPLLGFDARLCDEIYKMTTQGLTSLIQIPIGPYRHSPRLTGGLHG
>lcl|NZ_CP026520.1_prot_WP_042232258.1_693 [locus_tag=PC41400_RS03645] [protein=VOC family protein] [protein_id=WP_042232258.1] [location=complement(808212..808607)] [gbkey=CDS]
MGRLVHFEIHVDDMERAKKFYGEVFGWTFEDWSHYAGMPYFGAMTGDANEPGINGALMQRQGPPPNPDQPLNGYVCTLGV
EDYDSTEEKILNHGGKLALPKYALPGMAWQGYYIDTEGNIFGIHQPDKNAK
>lcl|NZ_CP026520.1_prot_WP_042232256.1_694 [locus_tag=PC41400_RS03650] [protein=FMN-dependent NADH-azoreductase] [protein_id=WP_042232256.1] [location=complement(808814..809455)] [gbkey=CDS]
MATVLYITANPQDEEKSYSMLVGKEFVQAYRSANPSDEVIHLDLYKMEIPQLDQDIFSGWGKLNAGISFEKLTRPEKMKL
GRLGELVDQFVAADKYIFVTPVWNYSYPPVMKSYIDSICIAGKTFKYLPDIGRVGLLNDKKAVHIQASGSFLSPGSDDAD
LEMGHRHLKTIMEFMGVPSFEGIFVEGMAAKPEQEPAIKEKAIRQAREIALRF
>lcl|NZ_CP026520.1_prot_WP_042232255.1_695 [locus_tag=PC41400_RS03655] [protein=carboxymuconolactone decarboxylase family protein] [protein_id=WP_042232255.1] [location=complement(809556..810029)] [gbkey=CDS]
MKSRMESPAIIMPDVSQALQALGKALFASAEKTAIPSRTLFLSYLRASQINGDSVCVELHSRNALAAGETTERLLAVAAW
RETEHFNDAERAALALSESVTRLSDREDPVPDKIFNEAARHFDEAALATLIAGIATANLWNRFNVSTRQIASVPAGK
>lcl|NZ_CP026520.1_prot_WP_042232252.1_696 [locus_tag=PC41400_RS03660] [protein=sigma-70 family RNA polymerase sigma factor] [protein_id=WP_042232252.1] [location=810221..811111] [gbkey=CDS]
MSKQVGLANTFEAHRKHLQAVAYRMLGSLSEAEDAVQETWLRLSREDGRGIENMGGWLTTVVSRICLDMLRTRKSRREDS
LETHLPQSFTSQENIIDPEQEAELADSVGIALLVVLGNLNPVERVTFVLHDVFTIPFNEIAPVVGKSETATRQLASRARR
RIRGAKTVPEADFAHQRDVVKAFLAAAHLGDFDTLVAMLDPDVVLRDDRQTGSSRVTRGALALAKQVSGRAQKAAQLALV
KGQAGIIVAPRGRLLYVLQFTVNQEKIAEVDLISDPERISRLDLAVLSDGTLVNPD
>lcl|NZ_CP026520.1_prot_WP_042232250.1_697 [locus_tag=PC41400_RS03665] [protein=RNA polymerase sigma factor] [protein_id=WP_042232250.1] [location=811378..811917] [gbkey=CDS]
METEYLKYFQEVDSTELYNLMTQYGDDVWRYAYALTGNYEQSNDIAQEVFLKVHLKMSTFRGQSSFKTWLFSITRNIAIN
EMRSGYFRRILLFGNVKSTQVAQSAEGDYMANQAAADLRKIIMSLSRKLREVLILNLEHELTIQEIAELLHISEGTVKSR
MSRARKEVERRWRNLNHES
>lcl|NZ_CP026520.1_prot_WP_241688782.1_698 [locus_tag=PC41400_RS03670] [protein=hypothetical protein] [protein_id=WP_241688782.1] [location=811907..812833] [gbkey=CDS]
MNPEKTNWAAMVKESSIGRGIFTEQMKQTVLTRVEQNQRKKLKKPFVYMAAPLTGLLICLLLFPVFDNLPAAFQTLGKAI
TGTNGVSSEDITLHYEPAKELSAIPDTDKGIRGYTLSRLPLASVHVKETVSIAGIGTYMDYIKSDEDPISYFGFQLFNHP
GSAGDEFYEIGYGKMTEAKLHKSNAFGLMDLRLDGKCGPERRCVYWIALKEGKVVAYEQMDAWTVYEQDLDKDGVTEVVA
STFAKEIYIYKPIKGQIQSVDVQAALKAGYGDTVTYNPDNQIFQITGKGGTKKYRYVTGADMLRQVHE
>lcl|NZ_CP026520.1_prot_WP_042232248.1_699 [locus_tag=PC41400_RS03675] [protein=hypothetical protein] [protein_id=WP_042232248.1] [location=813135..813497] [gbkey=CDS]
MSEKSWLVVQFPLKSVKGSQRDHWLKDKVMEYLEIFLADEGLGHVDGFDMGKCVSDPTSYALNIFCVVTDEARGIALIKK
VLREYRLDYTRIKIASMPYENGDAYTLKYAYKKGVTDFSL
>lcl|NZ_CP026520.1_prot_WP_042232246.1_700 [locus_tag=PC41400_RS03680] [protein=BlaI/MecI/CopY family transcriptional regulator] [protein_id=WP_042232246.1] [location=813653..814012] [gbkey=CDS]
MEPYKLFDAEYKFACLVWDHEPVNSTDLVKLSQTKLGWKKSTTYTVLRKLCERGILKNEEATVTAIIKKTDAQKYESRTV
VDKAFNGSLPQFLTAFLGEKKLSEKEAEELKRIIEEATR
>lcl|NZ_CP026520.1_prot_WP_042232243.1_701 [locus_tag=PC41400_RS03685] [protein=M56 family metallopeptidase] [protein_id=WP_042232243.1] [location=814009..815472] [gbkey=CDS]
MTSLFLTILNMSITASYVALAVMIARLLLKRVPKIFSYILWSAVAVRLLIPVSFTSSFSLLRLVQPQGKAGTGFMEFVPQ
SMGMQKNPMVDAGISGISRFINSTLPAAVPASSANPMQIVLWIASIIWITGIVVLLSYSVLSYLRIRARIRTATLVRDRL
FETDRITTPFVCGFLKPRIYIPTGLSEQELSYVLLHEETHIRRRDYWIKPLAFVILTLHWFNPLMWLSYALMSKDMEMSC
DESVVREMGDRIKGSYSTTLLSLAVRRSGVGTGSPLAFGEGNIKARIKNILAYRQPSPWMAAGSTLVIAALVLGCSANPK
PLAHPPQLSSQPLYSGYDMEKLMNNKTLYVGNHIKVGGLIGGMPRPNGLEGRGIELQTKAQPYGVTVYYVVNDDTAVLKD
GASSGEPFYRNSILMLSLIDNVDWITYSIADPAGPDDGSAYTFTLTREQAEKMLGEDVRSYGKDQTSLRQLIDRLERLNF
NETGSAV
>lcl|NZ_CP026520.1_prot_WP_042232240.1_702 [locus_tag=PC41400_RS03690] [protein=MerR family transcriptional regulator] [protein_id=WP_042232240.1] [location=815668..816429] [gbkey=CDS]
MYSIGQLSKKTNISIRTLRYYDEIGLLKPAKVADSGYRYYSGREIKLLQHIAAMKELGFTLASIKKMLSSGEETSETNWG
AYLDFELATVAEERRRLDEMEKLLQITRHAFEMKGDVESEDIFLFIRSLSDPSEGRETFLTQHFTDNEIRIIKSLPDLSA
DDPRNMAWAKLIREVKEHLHEPPSSAVSQKLAARIVEISMDWFQQDEQLIEKYWSLIRPEEGKEAKVYGMEADVMDYIDR
IVDWYLHHLEEGD
>lcl|NZ_CP026520.1_prot_WP_042232238.1_703 [locus_tag=PC41400_RS03695] [protein=SMR family transporter] [protein_id=WP_042232238.1] [location=816434..816784] [gbkey=CDS]
MEAKAEARMSEARAWAYVILGGLLEIVWASGFKYENVPGIVVLIALLTSFELIIKAAKVLSVGTVYAVFAGVGTLGTTIV
EIVMSGGAVSMVRIVFILLLLACIIGLKLSGKGSET
>lcl|NZ_CP026520.1_prot_WP_042232236.1_704 [locus_tag=PC41400_RS03700] [protein=multidrug efflux SMR transporter] [protein_id=WP_042232236.1] [location=816784..817098] [gbkey=CDS]
MDWLLLISAGLLEVIGVIGIKRVAQKNNVTNNIILIGGFIVSFQLLVRAMETIPLATAYAVWTGIGTVGAAVVGMLFFKE
PRGWLRIGCIIGVIFSVAGLKLAG
>lcl|NZ_CP026520.1_prot_WP_042232378.1_705 [locus_tag=PC41400_RS03705] [protein=hypothetical protein] [protein_id=WP_042232378.1] [location=complement(817095..817448)] [gbkey=CDS]
MVSGTAALGTEVLEARSGLEVRSDLEALALRLGSGVLALRLGLEALEPPSLVLSFSAKIIKEQLTRISLDNQPYLTQELL
TTHTPGSKRDLNFTTISCDRFAEKTTKGAAAAAPLII
>lcl|NZ_CP026520.1_prot_WP_042232234.1_706 [locus_tag=PC41400_RS03710] [protein=helix-turn-helix domain-containing protein] [protein_id=WP_042232234.1] [location=817682..817906] [gbkey=CDS]
MQVFQDWNQKVKKTFNATSNEVVLTVTEAGNLLGLSKDQMKTYVDKSTLTKVPIMRSVHRYLLLKKEIDELLER
>lcl|NZ_CP026520.1_prot_WP_042232233.1_707 [gene=msrA] [locus_tag=PC41400_RS03715] [protein=peptide-methionine (S)-S-oxide reductase MsrA] [protein_id=WP_042232233.1] [location=818000..818677] [gbkey=CDS]
MNDCHVGKETAGLTLPQSNLQTATLGMGCFWSPDALFGHLPGVIRTRVGYAGGRTEQPTYREMGDHSETVELEFDPGVIT
FEDILDVFWNNHNPVNINGYKGRQYLSLILYHNREQQDAVQAVISRRKKQGSTETETEVAPYSIFYPAEERHQKYYLKRF
PDAVDKLGTLYPSNHELVNSTLAARLNGLAKGYTSLERVAGEIKQWPVEPEARQKVLDLIRRIRW
>lcl|NZ_CP026520.1_prot_PC41400_RS03720_708 [locus_tag=PC41400_RS03720] [protein=arsenic transporter] [pseudo=true] [location=818900..820199] [gbkey=CDS]
MSTVVWACLIFLITLTLIIWQPRNLSIGWSACAGAAAALLAGVVTFQDVWDVTNIVWNATLTFVAIILISLILDRIGFFE
WAALHMAKAAKGSGLRMFIYVSVLGAFVSAFFANDGAALILTPIVLAMVRALRFDEKKVFPFIIASGFIADTTSLPLVVS
NLVNIVSADYFGISFMEYAGRMIVPNLFSLAASILVLYLFFRKSIPRRFDAEQVKRPADALKDPAMFRLSWVVLGVLLAG
YFVSEYLNIPVSVIAGVMAFFLLLMARKSSAVPVREVVKGAPRGDRLFFRRDVCGRIRAAQCRLNGSAGRCYSGGG*PGN
VRGDNGDGFSCRFDFFAHEQLAYRHD*RPRH*RNQRGGSRERSADLCECDRLRPGTQNYPYRFPGDAALAPCPEFQRSEN
LLGCLF*NGGSSYRSYVVYYSAGPVLLVKDSGL
>lcl|NZ_CP026520.1_prot_WP_042232231.1_709 [locus_tag=PC41400_RS03725] [protein=metalloregulator ArsR/SmtB family transcription factor] [protein_id=WP_042232231.1] [location=820259..820570] [gbkey=CDS]
MEPLLIYKALANETRRQILEWLKDPAEHFDPAPYIQHGIEFKTGVCVADMQGKTGLAQSVISSYLATMQKAGLVESERIG
KWTFYRRNEKTIREFADYIRNEL
>lcl|NZ_CP026520.1_prot_WP_042232229.1_710 [locus_tag=PC41400_RS03730] [protein=MFS transporter] [protein_id=WP_042232229.1] [location=820709..821896] [gbkey=CDS]
MKQRNTLLIVILALGVFGIITTEIGVVGVLPQITEKFGVTASQSGLLVSAFALVVALSGPFLTLLASGINRKRILLLSVL
VFAISNLVYMMTASYNVMLLFRILPAFFHPVFFSVALVTAAKLVPPEKSGQAVTKVFAGIMVGFAFGVPLTSYLAEQLSL
EAAFLFGALVNGAAFMGILFWLPSMPVKERMSYGKQLGILRKPGLWLNIAAVMCLFAAMFSVYSYFAEYLGQVTRMTGSL
ISAMLMVFGVVMIMGNFLFGVLLQKNVKKTVLLFPLLYAVIYLMNYWLGSYLVPMIGMIVLWGIVHSGGLIISQTWLMEE
SGEAPEFGNSLFVSFSNLGITIGSAAGGWFISRLGVHQLIWSGVLFAGLAFVLILIKMSAFRTQADSIEHQAATE
>lcl|NZ_CP026520.1_prot_WP_042232228.1_711 [locus_tag=PC41400_RS03735] [protein=hypothetical protein] [protein_id=WP_042232228.1] [location=822102..822752] [gbkey=CDS]
MKNWGQPLPEPAELKRQLLHMAALDVIMNEEDWLRVYRYVPDWKEDAALGIVDNGAGDDVYVVFAEEGVIMKGFDHMSPL
SPHAREEYGIWPGMYETAPAGLLAYLDNDLFEREDVTFCIWRQAGDSAWKTGEPQEAEGEDDGSDFLLGYLKHTPEEHAE
WAVSYYEKPVDLEAVRGLFAGAAVTEELIAALNPQRSAKDALTELEGLGLRADRGR
>lcl|NZ_CP026520.1_prot_WP_042232225.1_712 [locus_tag=PC41400_RS03740] [protein=hypothetical protein] [protein_id=WP_042232225.1] [location=complement(822888..823862)] [gbkey=CDS]
MYLPKWSGLYVHVKWFTEDHVWNPQPMAKVITPAFLLWLGVTLAALLVCVLFSESLERIDLVRRVHQFLNGLKRYQLLIL
RVGVGLGLLMQLFTGTYLAPSFVSDHVWVYAVLILAIAGLLHRRLLVVSGAALAVLYGYSLVTYGLFHGLDYFFYAGVIY
YLLVTESRWKQTATPVLYLCTGLSLAWLSMEKMTLAKLACSLMHEYGLPTLGFTVEDFVLISAFIEMGLAWAFIMGIMNR
FTAILLTSVFLMTTTFFGFKEIVGHTIMHTILILFIIEGSGAAKTLYQFHRTPVMRCLFVAVNFCIIVFGLMGIYIWLGQ
FNPL
>lcl|NZ_CP026520.1_prot_WP_042232223.1_713 [locus_tag=PC41400_RS03745] [protein=GNAT family N-acetyltransferase] [protein_id=WP_042232223.1] [location=824298..824705] [gbkey=CDS]
MSELNENLPEEKNSAPRETGEEGYVDIVVQKQGTSYILTGIGGTVGEMTYHMVDVDTWIVNHTFIDPKYRGRNLGRMLLD
YVVAEARENGRKIIPACSYVLEQFKQNPDEFADVWLKEGETQYSDTYSSDSVMKG
>lcl|NZ_CP026520.1_prot_WP_042232221.1_714 [locus_tag=PC41400_RS03750] [protein=metallophosphoesterase] [protein_id=WP_042232221.1] [location=824981..827581] [gbkey=CDS]
MRKERFFPGKRMTRSISLFVVFLLLALDLAGAVSASGTAGAGNGAQTEAVAVKETLAQWIFKDKGSNGVFPATGGVFQSA
SSIRDVGTNTDQYTYEPGENSVRNQGWHDGSGKKYWLAALSTKGYENITLSSQQTSSSTGPRDFKVQFSTDQQTWTDIPN
GKLVLAQNNFSCPNSTCKLTNLSLPQGAGYQDVLYIRWVVSSNASVSGGTVSSSGSSRIKDVLVTGERGSGEPGEQPTTA
VSRTPAPGAADAAVNTEISVTFNKAVVQDNAYKAAITDSGGASLSGVSSAVSGSTLKISHPDLTAGKTYKVTVPKELVKG
AQDGAAPSADITWSFTTRLPDSGVKTPSLLNMTFNGDPKTSISFDWYTAETVKGTVVQVVEASKAPDGTFPADLAQSYTG
SVTVIDTLMTAGDRSSKKYKKFASHKATASGLKPGTKYVYRAGNGDADGWSETGSFTTDRADNQDFHFLYVTDTQGSSKS
NFELWQDTFKRAIEKTVDPKFVLLTGDLTDDGDLEQLWQWFLGVPKKEFANVPFAPIIGNHEVEDYPNNNFYNHFNLPKD
VGTGAHDGSVYAFEYGDALFMQINSQYEGEVSPYKADPQFTKQLEWMRNQVAKSDKKWKFVSMHKGPYSSGDNATAEDDR
VKFYRKYLIPVFDELGVDMVFEGHDHMYMRSYQMLNNVPIKNVIIDEQGNVVNPKGTVYLMGNSGASKFYDLNPKADTFF
AAKNEQPNKKMFVDVSITGDVLKFTSYTAVKDKPIAVYESYSIKRTDGKPAKVENASAAKQSGKAVISWKAPAGTSEPVR
GYRIYEKNGRAGINWSAYVPAVSGQTAYSYTLNGIDSAQTYEFVVKAVGGRNNSEPVEISTAGGTK
>lcl|NZ_CP026520.1_prot_WP_042232219.1_715 [gene=thrS] [locus_tag=PC41400_RS03755] [protein=threonine--tRNA ligase] [protein_id=WP_042232219.1] [location=827957..829870] [gbkey=CDS]
MNIQVKLPDGSVREYSRGTTIEEVAQSISPGLRKNAFGGKLNGKPVDLNTSIDRDSAVEIITPDSAEGLDIYRHSTAHLM
AQAIKRIYGSKAVKLGIGPVIEDGFYYDLDLEKPLSTEDLAAIEKEMEKIAGENLPIVRRVVSREEALRIFGDMEDPLKL
ELIRDLPEEAEISVYEQGEFLDLCRGPHLPSTGRIKAFKLLNVAGAYWRGDSANKMLQRIYGTSFPKKAQLDEHLHLLEE
AKKRDHRKLGKELGLFMFSEEAPGMPFYLPKGMTIRTELESFARELQRQREYDEVRTPFMMNRRLWEQSGHWDHYKDNMY
FTDVDDTSFALKPMNCPGHMLIYKNSLHSYRELPIRLAEYGQVHRHEMSGALNGMMRVRTFCQDDAHLFVRPDQIEEEIG
RVIELIDHIYGVFGFEYKIELSTRPEDSMGSQELWDQAEASLKNVLDNRGIEYRVNEGDGAFYGPKIDFHILDALKRSWQ
CGTIQLDFQMPEKFDLTYIGEDSLKHRPVVIHRAVYGSVDRFIGIITEHYSGAFPLWLAPVQVKLLPVSDHYADYAMLVK
KSLDERGIRAEVDLRNEKLGYKIREAQLEKVPYMFVLGENEKSTGTAAVRKRGEGDLGPQSIEEICGLILEEIRSKA
>lcl|NZ_CP026520.1_prot_WP_042232217.1_716 [locus_tag=PC41400_RS03760] [protein=hypothetical protein] [protein_id=WP_042232217.1] [location=829993..831273] [gbkey=CDS]
MTERIPCKSADCGATILPRTAEKTGGYCMPCVQYRQRLEREEFIRRHRRDVNLYEGITDPVEILKIMCEPRVSDPLIRYT
AYEKTAEELYASLSAPDAERMKAYAAALIAEDEPDIAEDILLSLACFTGIPVQDCLPALIRKECYHPGILYKQAPAAIRD
FLLDQVEKDAENRNLLLLALAWIGDETVAARFNQWRETPPVWAEELYVPPHQYSLEAGWMLNGEGLRSDLFYKECRTFEK
GIPSGGEPVSLFGETEEFCRGCGGGMTVLFDLDLRHPSLRFLDMEGERLKIATCLSCTCYGFLFTDIDGKGGSGWSRYNP
APAVSPEPEGEGPPEDGERPALRFSGKERGTYYAAVWSQEPAVASQVGGHPTWIQDAEYPDCPGCSGKMSFLAQFSWEDA
DDYGEGITYAFVCKDCQIAATHFQQT
>lcl|NZ_CP026520.1_prot_WP_042232215.1_717 [locus_tag=PC41400_RS03765] [protein=hypothetical protein] [protein_id=WP_042232215.1] [location=complement(831632..832060)] [gbkey=CDS]
MRPKHILIAGTVAFAITIGAGFWNDESTVEVQIMIPDNPAAFDKTKTVATQTAAVPVQDELHQALGVTSDEEIYDALYNG
QSLAEVAASRSRDVRDVIALQTSQLSAQLDDRLAGGSLSREAYEAHKAELVEVITRSVTNAT
>lcl|NZ_CP026520.1_prot_WP_042232213.1_718 [locus_tag=PC41400_RS03770] [protein=LuxR C-terminal-related transcriptional regulator] [protein_id=WP_042232213.1] [location=complement(832549..835158)] [gbkey=CDS]
MTDTTPDAPVILKTKIATPLPKANLVARERLTDSLSRGINGRLTLVCAPAGFGKTTLLTQWAHSCGHTCAWLSLDARDND
FIRFWRTLIRSLADKAEDPAGQRLAALALSLQDISSSTFLDALLNELYEFPGKTVLVLDDYHLIGDARIHDSLAYFIEYL
PTALHVLIASRSELPFSTVKWTAKREQNELHTATLQFTADEAAAFYRHIHHHPLTDEQIRRLAAQTEGWVTGLQLALLSL
QSGTDAVRFLEEFHGDHRVISDYLFEEVASKLLPDLYSFVLKTSVLQEMDADLCDAVTESADSQVLLERAKKSNLFLVPL
DERGAWFRYHHLFAGFLQELLKRKQPEEWRRLNRLASAGCISRGFMNEAVEYAIAGEDYPRLESCLRTHMPAVLESGELG
NLLRWFRSFPPEYAVSPALSLHHAFVLLMAGHAEQAEQLLDRVESSLDASMEPEERKELQSGLLFVRSNLMFITGIFDKW
FAHIGTLLHEILPADPAFYTFNYNRTEPLVRRTPLGLKGVLSPDTERIGALFTSLLEARGWQGSYIYLYVIQSLSEGYYE
WNRLEDSRKLAVLVSRSAPAKELPGLYVPMHLTQAQLYAAQGRFELAHTALDEAFEASSGLKDSRWRQLLQAARIRLYLK
EDRSADVKKALPQLGVSGKDKPGFRQEFEYVTLARALGSQRKEKEALRLLELLKPQCEREQLLSGIIEIGILQALLENQR
GQRTTALRHLHEALELGARNGYIRSFLDEGKAMETLLRAYARQYRLTREVAGQGGGLHDYVLKLLSLFMPASPQRKTPAP
PLVEELSRSEISLLRLIRRGASNRQMAEQLTLSEGTVKVYLSRLYEKLGVSSRTQALVTAQELKLLDAE
>lcl|NZ_CP026520.1_prot_WP_042232208.1_719 [locus_tag=PC41400_RS03775] [protein=hypothetical protein] [protein_id=WP_042232208.1] [location=835351..835965] [gbkey=CDS]
MQAKPGGEYNGGQPSSFSNAVWDVHVNLPQQVENASVSIAGDMIGLLRLWEAFIELDTGAGSFRIPAGQIGIGAAAGQFV
PGVRLSEMTVSFEISMGRKERDALLTGEAGAGRYLLLSEPVGFAVTVSCGDQKMKSEDLQAIELALPLPEGLEPDRAINV
LMLDRKGSVRLLPARRISRKGRMMAAANGAGEGIFVLACTRGER
>lcl|NZ_CP026520.1_prot_WP_042232205.1_720 [locus_tag=PC41400_RS03780] [protein=GNAT family N-acetyltransferase] [protein_id=WP_042232205.1] [location=complement(836342..837655)] [gbkey=CDS]
MNVRLVKPDEMTEAATLADSIFRDGEQVSMAAGFPYLFSAVTSSSFGAYGEDGRMTAFMGLLPSAVRIGASELNVFSVGA
VCTAPEARGHGVAGQLLDAVKDQVAESDGDLILVSGTRSLYTRAGCREFGTIRRYTLTPEMAPALREAANEAAQAGVSDG
GTSSQTGADASACAASQAAAGASRGTDAAGPLRCREWTRSDLPALAKLASSREVAFEQSQWDLAMLIDAQAYARCVKLGH
RVYVAERGGVPAAFLVAALPGGLRPKMTPFAIEWAGEPSAVAALLAHALETAAPEKLDVLASRHDAGLAALLQKHAPHAD
ERILGTVYVPDAASLFHRLKPYLAALQPEKAALLEAGTGAAERGIRLSYPGLDDLELDADAWIALLFDGPFLLPGEEAGA
DANAAPVLPEGTPAQWREALTDLFPVPFPSTSGLNYI
>lcl|NZ_CP026520.1_prot_WP_009675332.1_721 [gene=murQ] [locus_tag=PC41400_RS03785] [protein=N-acetylmuramic acid 6-phosphate etherase] [protein_id=WP_009675332.1] [location=complement(837688..838602)] [gbkey=CDS]
MKENIHSLTTEQQNPRTRQLDQRSALDIVTLINQEDLTVAETVRQVLPAVAEAVDWITQRLSAGGRLFYFGAGTSGRLGI
LDASECPPTYGTDPKLIQGIIAGGPDAILKAVEGAEDSEQLGRDDADACGVSSKDVVVGIAASGRTPYVIGAMKRAREQG
AKVIALCNNHGTPMAAIADLAIEPVVGPEVVLGSTRMKAGTAQKMVLNMLTTASMVRLGKVYDNLMVDVQPSNYKLVERA
KRLIAMATGSDSEEISRAYAESQGHVKTAIVMILAKMSYEEASRRLQETGGKVRMAVESSESAS
>lcl|NZ_CP026520.1_prot_WP_042232201.1_722 [locus_tag=PC41400_RS03790] [protein=BadF/BadG/BcrA/BcrD ATPase family protein] [protein_id=WP_042232201.1] [location=complement(838668..839609)] [gbkey=CDS]
MDYVIGLDGGGTKTEALAIDLSGAELGTWTGGPGNPHAVTFATASAELGRLLTTVKDALPGRRCLAVCAGLAGVDTAEER
STMERYLTQTLTGSAYEGVSLLIKNDAEIALKAALGSSQGIIAIAGTGSIIYAFDSEGNKYRAGGWGHLLGDQGSGYDIG
LSVLQTVMKSYDGVLPATILTEAVLQKHGIRDASELRGIVYQPGVQKAQIAEHAEFAVRAAAAGDKLARQIIVRSAEDLA
DLVVALRNRHPELDRLKIGAAGSIFAYCPLFYDTFRVSVSQEDDLSDVVLSGRRSSAGAADYARELAASQGAK
>lcl|NZ_CP026520.1_prot_WP_042232199.1_723 [locus_tag=PC41400_RS03795] [protein=DUF1343 domain-containing protein] [protein_id=WP_042232199.1] [location=840123..841343] [gbkey=CDS]
MQGKVKTGADRFPEMAPALLKGRKFGLLTNPTGINRDFQSTIQVCAALDEAELGALFACEHGIRGERQAGVRFEDEIDPE
LGIPVYSLYGKHKKPTADMLSGLQAMVFDIQDLGVRFYTYLSTLIYTLEACAEAGVRVVVLDRPNPLGGLEWEGGLLDES
LLSMVGAWIMPARTGMTIGEFALLAADQMKTPCELYVVKMEGWEREMEYPDTGLPWLPPSPNMPTMDTVRIYAGHCLFEG
TNLSEGRGTTKPFEQVGAPWLKAEETCRELNALNLPGVRFHPVTFTPTFSKHAGQLCNGVMTFVTDPSVYKSAVTGLHLL
SCIQRLHPGEFEWLAPFKEGQSTFIDLLSGSDRIRLTLHEPGALEAIIASWERDGKEWAGIRSKYLLYGPEAAAAASAAE
ARGNGV
>lcl|NZ_CP026520.1_prot_WP_042232197.1_724 [locus_tag=PC41400_RS03800] [protein=serine hydrolase domain-containing protein] [protein_id=WP_042232197.1] [location=841333..842499] [gbkey=CDS]
MGYEAAASGASALGLSPEGVIAACRAVDREIAAGVTPGAVLAVVRGDEELVYAAGHADPDGPDPAPVHAGTLYDCASLTK
VTVTLPLVLGLIDEGRLTLNTPAASLLPEFASGGKEEVTVGQLLTHTSGLPPLVNLHASHLTREEMWQTVYRTPLQAKPG
ARMAYSDVGYMTLGRIAEEMLGVPLNEAARVKIFGPLGMDSAGFCPPAGRAADIAPTEYDASTGVRLHGVVHDENARALG
GVSGHAGLFAAAGDLVRYARMWLGCGLVPAPAAGARHDAAGAPGPRLLSRAAALASVRSFTESIPGASRGLGWALKGDPL
DASGDWMGPRSFGHTGFTGTSLWLDPERDLAVVLLTNRVYYGRGRTVAALRACVHNAIAGAVVTNDKF
>lcl|NZ_CP026520.1_prot_WP_009675323.1_725 [locus_tag=PC41400_RS03805] [protein=PIG-L family deacetylase] [protein_id=WP_009675323.1] [location=842544..843248] [gbkey=CDS]
MTEQKHHILAIGGHAGDLDLTAGAVIAKYTQAGHKATFLHLTPGEKGHPRLTPEEYAKQKIEEAHKFADIVGADVKFLAY
KDAELPVDEEVKYQVADVIREVKPDIILTHWKGSIHKDHANTHYIVNDARFYAGLKTIERELPSHYANHLYYADNWEDPY
DFNPEIFIDIPDEAYETWVKAMNVYAYARGETYGFPFIEYYKALTIVRGAPANFKRAQAFAVPTGALSKRVQFF
>lcl|NZ_CP026520.1_prot_WP_042232194.1_726 [locus_tag=PC41400_RS03810] [protein=MurR/RpiR family transcriptional regulator] [protein_id=WP_042232194.1] [location=843279..844127] [gbkey=CDS]
MNGGLVRLREILDDITPSERKVAEFVLSHPHRMLEMSVAQLAAESGGSQAAIIRLCKSMGVKGYPELKLKVAGDLNQTEE
RYEYREIRPNDSIDRIIHTVSANNIQSIRDTVTILDADTVRKAVEALSRANRTYFYGLGASNLIAQDAQHKFLRINKTSY
SFTDPHLELTSAVMLNEDDVAVGISYSGETDHVIECLKEARLCGAKTISITKYGNNSVSSHADYPLYTSSTENEIRSGAM
SSRMTQLNVIDILYLGVASREYDKSVGYLEKSRDIIRRMTGR
>lcl|NZ_CP026520.1_prot_WP_042232191.1_727 [locus_tag=PC41400_RS30050] [protein=hypothetical protein] [protein_id=WP_042232191.1] [location=844245..844433] [gbkey=CDS]
MLSIHLERQALDALAFYQLRIEDDMVAHIPAFQQICEELRGLIAGLVFVLVDGRKGWRGVRF
>lcl|NZ_CP026520.1_prot_WP_042232189.1_728 [locus_tag=PC41400_RS03820] [protein=sigma-70 family RNA polymerase sigma factor] [protein_id=WP_042232189.1] [location=844748..845242] [gbkey=CDS]
MGLEELYKEIQPKMYAFFYVKTQSREAAEDLTQEVFYEAVKGHGSFRGDSTVRTWLFGIAKHKLSKYYRSKKYGQELAAK
LAAESRESVTPEKQVLDREDRDILLERIFRLDEGSQEIVTLRMYAELSFREIGHIIGATENSVRVRFHRIKLQLHKELEG
YYER
>lcl|NZ_CP026520.1_prot_WP_042232187.1_729 [locus_tag=PC41400_RS03825] [protein=hypothetical protein] [protein_id=WP_042232187.1] [location=845232..845819] [gbkey=CDS]
MNDKNTVRHIIEDLLPLYEEGLLSEETARWLEAQTADDADYEKLVRLSGQSLPKPELPEPAEDYAKMMAKINRKLSLYQL
LFMAISFVLAIRTSLLNESFGFVLWYAVLGFVTYLFYKQMKIVFFLSFAPVFLWSIGDSIYEIVNGSGDGGMGVFVFSPL
LVAVVTASIHCLFAFVGSLMGLLALKIRKMGEESE
>lcl|NZ_CP026520.1_prot_WP_042232185.1_730 [locus_tag=PC41400_RS03830] [protein=hypothetical protein] [protein_id=WP_042232185.1] [location=845816..846562] [gbkey=CDS]
MKTRKKKKGMYIALLVLLGLLGLLVGSLYNSLNGNPLSKWLAQRELQQFMVEHYPEKRLRIEKGFYDFKFKRYEFRVKEI
GVDADKGEPAEYEFSVGGWIPKVKVDGIRSANLDMPLMERLRGEAEAELQAAFARRVPAVREVSVQLEVVKGQLAENAAW
TPSLKLDEPLYIHVVTDAAGASKQQVAEAAAEIAAVLDAGGYDYGHVTINSNVMDDEAGKDPYGYVKYVARFDKGGKVSA
GDVEEVTK
>lcl|NZ_CP026520.1_prot_WP_129112450.1_731 [locus_tag=PC41400_RS03835] [protein=hypothetical protein] [protein_id=WP_129112450.1] [location=complement(846936..847277)] [gbkey=CDS]
MKKIIKKVSTSLLAIGLSLTMFTTGALASENQKLTSITSQDEVNPFVGVDYSNLKKGHSILLGGSSFSIVYGSNYVSVSP
TGWTTGLAAGDARVDAYDANGNHYWTFYIHVTN
>lcl|NZ_CP026520.1_prot_WP_009675348.1_732 [locus_tag=PC41400_RS03840] [protein=carbohydrate ABC transporter permease] [protein_id=WP_009675348.1] [location=complement(847559..848422)] [gbkey=CDS]
MAAKAAKLRPAGAGTSGSSPFFQGLVFVLLALGGIVFLIPFVWMILSAFKPESEVLRIPPTLFPETFTMENFKNLFERMN
FGVYLKNTLIVVLFSFVGLFLNAMAGFGFAKYRFKGREGLFYLVLATMMIPGQVTMIPVYLILNSIELTNTLVGVVLPGL
VGAFGVFLFRQFMSTIPDELLEATRLDGASELRTFAQIVLPISKPILSVQAILTFIAGWNSFLWPLIIANDEKLYTLSVG
LSLLKGQYDSNFALQMAGSAFMVVPILIIFAIFQKYIIDNYSISGMK
>lcl|NZ_CP026520.1_prot_WP_042232183.1_733 [locus_tag=PC41400_RS03845] [protein=sugar ABC transporter permease] [protein_id=WP_042232183.1] [location=complement(848443..849330)] [gbkey=CDS]
MKGFSNSKAPYFFIAPTLILLTVFSLLPIFIALIISFTDMDLAGLADYSNINFIGFKNYISVLQDPIFIKSITNTLFYVI
IGVPCVVILSLGIALLINYGASRVFKVFRVIYYLPSVTNVVAVAVVWSYLYNPAFGLFNYVLETAGFDKVPWLTDPLIAK
ISLILLALWRAVGLNMIIFVAALQGIPKTYYEAAQLDGASTWKQMIHITVPMLRFAIFFVSITTMIGWLQFFEEPFVMTQ
GGPLDGTTSVALFIYNNGFKLSNFGYAAAGSFILFVAIIIITLIQFRFQNKDTDM
>lcl|NZ_CP026520.1_prot_WP_042232181.1_734 [locus_tag=PC41400_RS03850] [protein=sugar ABC transporter substrate-binding protein] [protein_id=WP_042232181.1] [location=complement(849440..850702)] [gbkey=CDS]
MKRKNWAAGIATTVLLTSVLSGCGGSNAGGSDKGSGESGGKVTLNVWGMGEEAKSLPKIAEQFEKDNPNIKINVQALPWA
SAHDKLLTAVASKKGPDVLQMGTTWIPEFAAAGALMDLKPMADKYPEIKEDNFYPGSIQTTKYEDKTVGVPWYIDTRVLY
YRTDLLKEAGYNEAPKTWEELQDAAQKLTKRGSGKYGISLDPKEQSLTFMFARQNGSKLLDGKQPLFDQPQFVEAVDYLN
SFFKNGSAPKDLGIDAVQGLSGDGIVPMFISGPWMVKLINDKAPDLKGKWATAVLPKKENNISALGGSNLSVFQHTEHKE
EAMKFLAYMSKPETQLKWLEMTQSMPAAKKAWEDPVLKEDPNYKVIGEQMKASEPMPMIKPWEEIAQAYLKSFERIYRGQ
AEVKPELDAFNKQAQQILNK
>lcl|NZ_CP026520.1_prot_WP_042232179.1_735 [gene=bglX] [locus_tag=PC41400_RS03855] [protein=beta-glucosidase BglX] [protein_id=WP_042232179.1] [location=851051..853210] [gbkey=CDS]
MTEQQLLARIKEMTLEEKVAQLLQLAGPFFEGADGQITGPMASMGITGEMTAHAGSVLGVAGAGQVKAVQDAHLRGRPEA
IPLLFMADVVHGYRTIFPVPLATGCSWDMELVEKSAEIAAREAAVSGVHVTFAPMVDLVRDPRWGRVMESTGEDPYLNGE
YARAFVRGFQGADLTGDKERVAACVKHFAAYGAGEGGRDYNTVDMSERQLREQYLPAYKAALEEGCEMVMTSFNTVDGIP
ASGNKRLMRDLLRGEWNFDGVLISDWGAVKELIPHGTAADEAEAARRAIEAGVDIEMMTACYVHHLPALVASGEVDEALI
DEGVLRILRLKQKLGLFDDPYRGADEEREREVVLCADHRRAARELAQKSCVLLKNEGVLPLKPDRSIALIGPFAESGDIL
GPWSWLGSQDEAVPLYDGLAAHADPSRITVAEGCGIESGTPEQTAAALEAARAADVIVLALGEHSGMSGEACSRGDIRLP
EAQLRLVAELRTLGKPMAAVLFNGRPLDLHGLIDQTDAVLEAWFPGTEGGAAIADLLFGRVNPSGRLSMSFPHSVGQVPV
YYNHFNTGRPKDALDAQEHYVSKYLDIPNEPLLPFGFGLSYTTFAYDTPTLDSASMSAEQPLTVRVNVANTGETAGEEVV
QLYIRDMAGEVVRPVKELKAYRKLFLKPGETAEVRFDLTEEQLRYHHSDLSFASDPGEFAVFVGPNSRDTAELRFRLTK
>lcl|NZ_CP026520.1_prot_WP_042232177.1_736 [locus_tag=PC41400_RS03860] [protein=cellobiose phosphorylase] [protein_id=WP_042232177.1] [location=853232..856663] [gbkey=CDS]
MTQLKDRSLQLKSGSLQFSFLNSGDIAEVVSGSTLINQWIGNPLDGSMNNLYLRLHREKGIQAYPLLGVRSASLISRKES
GIVWNGEVEGISYEVTFTLTPQNIWFWDVLLTGGTGDTPVDILYGQDLGLADKGAVRSNEAYMSQYIDHSVYQDEKRGYV
VCSRQNQPQGGAFPYVQQGSLTGADGYSTDGFQFYGLSYKETDKPEALARPHLANEIYQYEFAYTALQSRRVISGAEPVR
FIFYGLFKENHPEAVTALEFAEEVSAAWEQAEALNAVAGGNGDSGSGTDTGTGIGTRTGTTALEPVRLAGDIGAPLTGLT
MTADEVAGYFPDRRLEETLEGELLSFFTETNEHVVLKAKELRVERPHGHILMSGDNARLRESVLTSTAYMYGLFNSQLTV
GNTSFNKMLTNARNALNVSKTSGQRIYIETGGEYRLLTMPSLFEIGFNYTRWYYKLADDTVVVTGFTAADAPEVRLQLRS
VQGKTYRYLVTHQVSMNNNEYEAPFHLSVQEDGQTLLFTADAASDSAQVWPDLTYRMQVAGAPLRVTDETRLAAGAEPGS
ASLAVLELGESSSWSMTVQGLLRGEELSFTDREADLEAERYRAFLGSVMNGFHLYRAGGEGDPELAKMNVLAWWYTHNML
VHYSVPHGLEQYGGAAWGTRDVCQGPSEYFLATGKYGEVRDILRTVFSHQFEDDGSWPQWFMFDRYSHIQQEESHGDVIV
WPLKVLADYLSATGDLSILDEQLPYTRRHRGNFTEETASLLEHAVKEIGYIRSHFLHDTHLSSYGDGDWDDTLQPANAQL
KRFMVSSWTVSLTYQVFRRLSVVLADSRSALAAELRGMADGIQADYMRYMLPSEASVIPGFLYMEEPGKAEKMLHPEDTK
TGITYRLLPMTRSMISELLTPEQAEAHYRLIRAEMYCPDGVRLMNRPANYAGGVSTNFKRAEQAANFGREVGLQYVHAHI
RYVEAMAKLGKTGEVWRGLQMINPVRLQDVVPNAELRQSNAYFSSSDGRFSTRYEAQERFNELRDGSVPVKGGWRIYSSG
PGIYMNQLISNGLGIRMDAGSLIVDPVLGEELDGLCFQFRILDRPVTFVYHLSGGEVSHVTINGREAAARRLNNRYRTGG
LLLSGEQLESLTAEGPNVIEIYS
>lcl|NZ_CP026520.1_prot_WP_042232175.1_737 [locus_tag=PC41400_RS03865] [protein=LacI family DNA-binding transcriptional regulator] [protein_id=WP_042232175.1] [location=856824..857789] [gbkey=CDS]
MVSIKDIAKKAGVSISTVSYALNGSPKVTDETCARILAIAKELNYTPNAAARTLKKRESKIIGVFLTDFSGPFYGHLLQG
IKEYLNGHDYELIVCSGSQSHRFLPERMIDGAIILDAAFTDEELLQHADRGHKIVVLDRELIHPNINQVLLDNKAGANLA
ADYVLEKGHRKLYVVAGPQDSYDSDKRLHAVRQTVGRYKDVDCTELPGSFDKESGEKAARRILEAYKEPAAVLCLNDEMA
VGVYNTVAESEYRIGEHIHIIGFDNIELSAYVSPRLATIDYSKHKWGALAAEQLLKLIAGEKVEHERIYVTLVKGESVGD
V
>lcl|NZ_CP026520.1_prot_WP_042232172.1_738 [locus_tag=PC41400_RS03870] [protein=copper amine oxidase N-terminal domain-containing protein] [protein_id=WP_042232172.1] [location=857950..859575] [gbkey=CDS]
MLIRKLLKLSLLLLLLFGVSISFHYQGHVSAESAGTNRIITLTIDNPQADIDGQKTLMPAAPILMKDTSMVPFRFLGELL
GAEVGWENSTQTVTLKGTGQTVTLSVGQREASVNGVSTILDQPAAIVNDTTLVPLRFIAESLNRNVAYDNASKVITIVPK
EETKPVPPVPEKEAPKKKLEHPTVDNLSLKIGKSYSKICINYLECKSAKTRFVSSFVTDSNDNIYMIDYDAKRQIKYEYL
ISMYNQKTGDNRKLEQVLNVNEKFDIEYLDKKNQLQSLSYRDLAPEKIFYDEKRDKLYMMASVEKKDIKIVIYEILPEVK
LITYNADETLFDEENDIMTTLDGVNLIFSNTFFGSIYSVEAGKERSLKAVIAAEKKPKLVSLIKDNKIYLMDYINKTIHM
MREDGRLIEVAKVKMEKIGAAASKGEFFYAADKNGFYKIDVTGKVEDYVKLSELRYNSGLYEPKTKTYDQLVPGNMNFED
LPYSPLIQRGPGGEVVVGTLEMDFFDDTIPVDFAVDSQGNIIIHDNVHKLLRRINVYGYEQ
>lcl|NZ_CP026520.1_prot_WP_042232170.1_739 [locus_tag=PC41400_RS03875] [protein=TetR/AcrR family transcriptional regulator] [protein_id=WP_042232170.1] [location=859760..860338] [gbkey=CDS]
MARQRAFTKSELLDATERLLMERGYDGFHLKALSEMLSGARSTIYEYYANKEEIVAACMRRTMEQTIEACEGIEQLEPVA
AIKKVLLIFLELSNFHTLMLAAPKIDETASDKARRDLEFLNEGHADLKKKLTALFQKAQEAGALRADIPLPVITAVFFHA
IDTPNWLNVPTEEWADMLFGLWWNGGGNPLTK
>lcl|NZ_CP026520.1_prot_WP_042232168.1_740 [locus_tag=PC41400_RS30535] [protein=S8 family serine peptidase] [protein_id=WP_042232168.1] [location=complement(860588..863860)] [gbkey=CDS]
MKNRHLIQKLSVMSLAFAFVAGTALPHASAAQSGKSIQPRTQLNGSLNPGTNSLPGQVKGYISPKLNTASTNSVRVIVQL
QNQPVAVGKYAAKLGMSALAAESTESAITQEQTTLVSVAKNSGINLQVNYQYNTVLNGLEVTVPANKIPALAKLAGVKSI
HPSANYFPIPVDEPKGLDVSSPNYDITPLKQIGADVAWAKGLTGKGLKVGVIDTGVDYLHPDLKDAYKGGYDSFEKDNDP
YEEPPLTPEEDPYGTGFEGTSHGTHVSGTIVGRASNPSSDIVQKGVAYEADLYVYKVLGRNPETGRASGSSAQVIDGIEH
AVKDGMNVINLSLGSDSEKDPNSPDVVAINNAVLGGVTAVIANGNAADKGPYYYSLGSPATSQLAISVGAVTSPKTQYTA
SVSEAVYTPLVSPGAAAVPPVTPAAAGYNLNVMAWETGQENFASILGTNPRDVVYVNLGQPQDYEGKDVTGKVVLISRGN
LAFVDKIANAKQRGAKAAIIFNGNTERGDATKADLSESVIGRDGFVGSAAYLGDGFEYIPTFDMKGTEGRALARKALAAQ
GQTLKLAFGANYPKSEIPGDTMATFSSRGPNVDGDLSIKPDISAPGVSILSTWPAYGKQKPNASYNTAYSRISGTSMATP
HVAGLALLLKQQHPDWSPLDIRAALANTSDKISDLDGKLYDVYSQGAGRVNVGTAVKTQALLQSVEPITILDKYWQPQAV
TNYNPSVSFGLLAPGTNAQKELQLKNVSKQSVTYSAKVVLHPNVTSDPNAPTQTPDVTNLTAQLLGLQNGKLTVAAGASK
GFYLSVTPKNNAVKGVYEGEVLLERSGQPSLHLPFAVHVGQDKPENGFGLQELEQTENAIYPDNRDGSPTTTDLSFRLTA
KDVNAISLWAVGLDDKEIGILDEITRTDANGRLKLIEPGVYSFEGIDGSYVDGELDANGNLKVKQLAEGTYKLSVIAQKI
DEFGNNVDGISYTAYSSLRIANLENDRVAKAKDDFTAKIVNTRKINQPVLQLPKTDGIVYKVTNSSNKAYIDNSGVLKFV
PNSGIVNVELTVTLSSVKNPAVKTTVKVPVVLSKSSIPQRQTADESALAE
>lcl|NZ_CP026520.1_prot_WP_042232166.1_741 [locus_tag=PC41400_RS03885] [protein=helix-turn-helix domain-containing protein] [protein_id=WP_042232166.1] [location=864139..865008] [gbkey=CDS]
MDARLHDLFRPVQANGRVSTYIYEEKGPSPALKPYVCCYWQSEPRGSGSAQLLLRPDAVDRVLPDGCTDIIFTENLKTGE
MDVRYCGTFQQAFTLSYEEGAPMRSFGIRFMPGGAYPFLRTALSEFHDQAVAARQIGSVIGRETEERVREARSFGERIGA
VEACLMRQLSDKGTDGDNRMKNVLYRVFISGGGETVSELARSETVSTRQLHRSFNQWVGLSPKKFAEVVRFQRLLRDIQA
GSVTDWAAAAAERGFFDQAHMIREFKRFYGASPIKAAAEYRGMSDLSYL
>lcl|NZ_CP026520.1_prot_WP_042232164.1_742 [locus_tag=PC41400_RS03890] [protein=DinB family protein] [protein_id=WP_042232164.1] [location=865071..865559] [gbkey=CDS]
MSGVLQIRDHLLDELELSVRTSEALIRRIRPEEWAFTPGENMRTLLQLVHHIAALPASDLAILQEKSQPEVELVENGAQD
ITDPEQLAKRLRTHFDTLKAYMVSLSEDELLNKETKAFYLEHSMPQIKWLIEIVTHLFHHRSQLYNYLKQNGHELNFFML
YA
>lcl|NZ_CP026520.1_prot_WP_241688783.1_743 [locus_tag=PC41400_RS03895] [protein=hypothetical protein] [protein_id=WP_241688783.1] [location=complement(865694..865831)] [gbkey=CDS]
MLEDFVSVEAEAAAGTALLGCCICCTVYFPYKQLLARRRLNRSRS
>lcl|NZ_CP026520.1_prot_WP_042232162.1_744 [locus_tag=PC41400_RS03900] [protein=glycerophosphodiester phosphodiesterase family protein] [protein_id=WP_042232162.1] [location=866051..871477] [gbkey=CDS]
MRKFSRFLSLFLTAALLCGTAPLGLFDAAAYAADRGRKALDVRKTLTPPIIDGKLDESLWNVSETMNVRTGEGTFKNARF
GMLWDNKYLYIGVKAEDDTLFSSGTGYWFEQDNINFFFDPTLHQSAPFAADDMQAGFVYKPGSTPEFHFGAALGNHSGKD
EKKILRAVGTDQGGWTLETAIPWEMLRFDAGLQKQLGFEIGVTDRYEADTSKQRTSYWSAYNSNSFWNDTAGYGILNLKD
ENPVSAQLDPVLLQDNFDGYPAGHTPAGWISDTNAGSDPFTVVQDTYGNGRMAFDGKASGKQARISAPVQWDDYIVEADM
RFEEVLNDARWAALMFRGAADGKNPYNQMAVRRNGTFELAYRMPNNNWASPTPVSGMWKPLEFNKDYTLKVRAVGNNVKE
YIKAKDEADFTLLMDQNFSSNLLEKGKVGFQADQSKVSFDNLKVTRVTADRLDVTMPGTLEALTGPATVTGSVYYSDGLT
ETPAPGAIRLYSSDETIVKIVNNQLYPVKAGKVTVKAVYANAEAAREITVTPSATGAQAVKLQHAEGYVLTDTGVKLDLN
TLSFQAEFSDFSKGTLKGGELQWKPAGPEVTAANGTLQVNKKGAYPVTVQKDNASLTLMVIAKNPGEKEYTLYEENFDQV
AEGALPKDWTRKEGTTASAAAVKSGAFELNASAAPDNPSRVLLPEYLGLFGNYKIEADMTHLSANDAARWHSIMFRIQND
NVPYYQMAVRQNAAAVNGVEFAERTSANEWNVMDRGSYTEAIDPGKMYHYTVKAYGDRVQEWIGDKLLVDTDNAGAYKKG
RIGLQANGSRMKVDNIRVTLQQDALPPLPSERFVQVVEPETKISLAPTVVTELKSSADLVKLQGTSLPATAMMYVNGNLK
ITAPGGAEIAGLDEILGALNNKVIPAFYVRDELSTDKLIEALKTRGIEDAFIVSDKGELVKKARLAYPIIRGILDFRQAK
ADTNEDLMEIRRKTTVNLAKIALLSQNAATRESMAYLQQRTLVVWAQDQSVQADKTVSLHNLITGGANGIVTDAPEAAIG
ALKLYSNGTTLIRKPYIIGHRGIPSEAPENTIESNALSLEYGADYIENDMYLTKDGHLVIVHDSVLENTTNGKGKVEDFT
LAELKKLNANKPFPTGYPDVKIPTLDEQIDLARSKGRMVMAEIKTSTPAAVDAFVRIIKEKKAEDIVDAMSFDTNQLKRL
AELMPEMPSGLLTSGYANEANVNKSLRETLKVLQPLNATFNTSYGGLGKNFMEAAKHRGVIISPWTFNNKNDFMKFFGLG
AFGITTDYASWAADWAAFVKPGKSAYTLSPGENVALSLEVESYRGTKTNAAPDIVLLDGQDVVEVSGSKITGKKAGKAHV
LLRYTASMDANNKYDLYTQPVTLEVKGGSGGDNGSGGNNGNNGNNGNNGGNNGNNGNNGGSAGKVTASGGAVDASALREA
LKTHSTVEIAYTGDTVEIPASALMDAPQGSRTLVIAGPVGKYTVQLADLKLDEAARKLGGESRSMTVRYTLRYAAGSDAA
AIGAAVKAAGLQTAGTPVQLEIAAKAQSGGSAALETQGQLELTLDGTFDAKRATGVRAYPNEKKVSFAPSLFAPAGTAKT
AVTFKRSGSGFYAVVSGAPVFADMAGHWAQADVELLAAKLVVEGSGGGRFDAERSVTRAEFAALLVRALGLPKAAGPAGF
GDVGSADWFAGDVAAAAAAGLIGGYEDGSFRPQRPITREEQAAMLVRALPYAGAASAAAITKPQQEEILARYTDSGKIGW
AQAEVAAAVHAGLMNGMTDGTLAPASEATRAQSAVMLRRFLVTAGFIN
>lcl|NZ_CP026520.1_prot_WP_042232160.1_745 [locus_tag=PC41400_RS03905] [protein=hypothetical protein] [protein_id=WP_042232160.1] [location=871756..872502] [gbkey=CDS]
MLKKTSFLAALLLIMAVVSFTGAAMARDYLLPHSAYIEICEEDPGDQLCQNFCEDFPDTPVCSGQESEGQETGEYVVEAR
LDVQFNRMNIKVNGKEFVAPNILYEGTTYLPIRDLANLLELDVNYYGAAQTAYIGQLPAGEVPDEVIDEWEKQLSEESAN
GDAAAGTLVRGGGTIDVYLDRVQVKVHGERIVDSTFLYDGTTYVPMRAACDILELPVNYYGPTSTAYIGEVPAGEVPKAE
VDKWKSGK
>lcl|NZ_CP026520.1_prot_WP_042232158.1_746 [locus_tag=PC41400_RS03910] [protein=DUF898 family protein] [protein_id=WP_042232158.1] [location=872546..872917] [gbkey=CDS]
MANTTINVTLPKDLDFGSNPKPQGGASYFDGGLLELIGLYLLGAIITALTAGICYPWSLVIIYRWKIEHTVIEGHRLRFE
GTAVGLFGSWIKWFLLGIVTFGIYFFWAGIKLEQWKTKHTYFR
>lcl|NZ_CP026520.1_prot_WP_042232156.1_747 [locus_tag=PC41400_RS03915] [protein=M4 family metallopeptidase] [protein_id=WP_042232156.1] [location=complement(873092..874669)] [gbkey=CDS]
MKKKLTSLVLGGTLLIASALPASAATAEVATPKSVTPKFIGETWKAPAGLSNDEKVWGFLDHKKEALSLSGSLKDNLKIV
TKQTDAASGTNHYRLKQYVKGIPVYGAEQTVHLDKNGNVTSYLGSLLADSEQEVANGVTAKISASDAIKTAIADTEKAVG
KLGQQQKPAQAELTIYPLEGKNVLTYATEVNVLEPSPIRTRYFIDAVDGSIVKKYDLLDHAVGTGTGVLGDTKQFVTTQS
GSTFQLKDTTRGNGIETYSANNGSSLPGVLKTNTNNTNWTDGAAVDAHAYAEKVYDFYKTKFGRNSLDGNGKLIKSTVHY
GKNYNNAFWNGVQIVYGDGDGVTFLPFSGDPDIIGHELTHAVTEYTAGLEYYGESGALNESISDIFGNTIQGTNWLLGDD
IYTPGTPGDAIRSLSNPEQFNQPSHYSNRYTGTSDNGGVHKNSGINNKAFYLIAQGGTHYNVSVTGIGRDKAINIYYHAL
TYYLSPYSNFSNIRAAAVQSAVDLYGAGSAEAVSVGKAYDAVGVN
>lcl|NZ_CP026520.1_prot_WP_042232155.1_748 [locus_tag=PC41400_RS03920] [protein=alpha-glucosidase] [protein_id=WP_042232155.1] [location=875255..876874] [gbkey=CDS]
MQKTWWKEAVVYQVYWRSFYDTDGDGYGDLEGVIRKLDYIKELGADVIWLNPIYGSPDVDNGYDISDYRTVMAKAGTMET
FERLLDEVHARGMKLIMDLVVNHTSDRHPWFTEARSSTDNPKRDYYIWRREANNWRSYFTPSAWEYDEQTGEYYFHSFAT
EQPDLNWENPRVREEIYEMMRFWLDKGIDGFRMDVINLLAKQAGFPDAEHPENISYLGNNPGIHEYLQEMHERVLKHYDV
FTVGEIPFVTPEDGLLYVGEERGELNTLFHFEVCDYMAHWEMDKFKAIQKRWYDGMWGRGWNSQFLNNHDHTRLVTRFGN
DKEYRVQSAKCFAVLLHTLPGMPYVYQGEEIGMTGVRFDSIEDYNDIALKNKYKEEVEKGRDPQEVFRSLLHLARDNSRT
PVQWDGSANGGFTEGTPWIKVNPNFREINVEQALDDPDSVFYTYQKLIRLRKENPVMIYGTYEPLLEEDDRIYAFLRELD
GEKWLILLNISDQSTVCTLPGSIASAEKELILANYKVNGPDGSSCEMKPYEARVYRIKS
>lcl|NZ_CP026520.1_prot_WP_042232153.1_749 [locus_tag=PC41400_RS03925] [protein=NAD(P)-dependent oxidoreductase] [protein_id=WP_042232153.1] [location=877032..877895] [gbkey=CDS]
MNVFVAGATGAIGRFLLPLLVKEGHTVYAMVRSEAGKEAVLAQGGVPVEADAFDREGLIAQLRRVRPEVVIHQLTALSGY
NLEENARIRKQGTRNLVDAAREAGVRKMIAQSISWMYEPGEGPAGEDVPLDLEAPEPRKTTVDGVAALERAAAEMPESVI
LRYGLLYGPGTWYARDGAVAGKVRSGEVKATEGVSSLLHVEDAALAAVAALGWPSGAVNIVDDEPAPGTLWLPAYASALG
APEPEFQPGQGRGERGASNAKARREYGWEPVYPSWREGFKASLAQGN
>lcl|NZ_CP026520.1_prot_WP_129112451.1_750 [locus_tag=PC41400_RS03930] [protein=hypothetical protein] [protein_id=WP_129112451.1] [location=complement(877892..878320)] [gbkey=CDS]
MGTVRFRQHEDKAGRRVPVGFGLAGMDATLASPSGRLRLWPGGSRSVRAAFNPARCPGLAALGWTNAVPGSSRLDRRPTP
QLAARANAAPVGGLRVRPGPPRRGPMHCAPRAGRDAETNRTAAQTKRTGGEHTSGPSINWRA
>lcl|NZ_CP026520.1_prot_WP_042232148.1_751 [locus_tag=PC41400_RS03935] [protein=hypothetical protein] [protein_id=WP_042232148.1] [location=complement(878308..878724)] [gbkey=CDS]
MYSPPYPVSYPFCPPVFVSAVHGLPAGPGLYPRFPENPGPGYPPVEPKQLISSATSFRKLLADGNVVLARLSDEPFARRL
MIAAQAGRKQEVDRLMKDIAISSVLSARYTPSGLIVTVTPGVQDPACCVLTMSLKWGQ
>lcl|NZ_CP026520.1_prot_WP_042232146.1_752 [locus_tag=PC41400_RS03940] [protein=DUF4097 family beta strand repeat-containing protein] [protein_id=WP_042232146.1] [location=878934..879896] [gbkey=CDS]
MRRNMGFLLIGAGVLALLYYFNPVGVFSWNGLSFKTQDIHLEKTAEAASLQNITVETGSADTRIVKGEGDQVNVRLDGNV
SKKQAEKFQLKVDAQGDTLSVGIEEPNEIGVGVTIIDVELVVELPEKNWKSVRVRSGSGNIDLDSLKGDSVETKAGSGDT
SLREIKFAKLVANAGSGNINVSDVQGDAITLKADSGDIRAEQYAANDLKFEGHSGNVELKQGKGSLQGETHSGDIRVEAG
ELLRDADFKSGSGNVKIELIEEPKSLAVDFEGGSGNGSIDSENFSYEEKNEKKNRLKGKFGSGEVKLKVRTGSGDFQLTK
>lcl|NZ_CP026520.1_prot_WP_042232143.1_753 [locus_tag=PC41400_RS03945] [protein=GNAT family N-acetyltransferase] [protein_id=WP_042232143.1] [location=complement(879960..880409)] [gbkey=CDS]
MKEKKVREIRLPDYPAIYLLNQDFNPNLHSFSEQKVKEKIEIITQKTNDIIFVCEQDEEVIGYIHGSPYELLFSDSLVNV
LGFVVKKTFRNQGAGSMLIESLEAWGKTNGFYGIKLLSHPSRTAAHRFYEKRGYTFTKDQKNFIKTFDC
>lcl|NZ_CP026520.1_prot_WP_042232142.1_754 [locus_tag=PC41400_RS03950] [protein=GNAT family N-acetyltransferase] [protein_id=WP_042232142.1] [location=880518..880802] [gbkey=CDS]
MRETIKRGNGYVVMENGEAVAEITYQPLGSESLVIDHTFVSESLRGQKVGEELVRAVVDQAREEKKTVVPACSYAHALFK
RHKEYHDIWRREEE
>lcl|NZ_CP026520.1_prot_WP_042232139.1_755 [locus_tag=PC41400_RS03955] [protein=glycosyl hydrolase family 18 protein] [protein_id=WP_042232139.1] [location=881159..882916] [gbkey=CDS]
MKKILTGPKSVVKKTAGFLLTATLFASLIPAMASAADRGAWAASTSYAVGDTVTYSGKSYQVVQPHTSLNGWEPPNVPAL
WKEVQGGGTVDNQAPTAPTGLKVSGTTTSTSITVAWNPSTDNVGVTGYDIYRDATLVGSVSGSTLTYTNTGLTPNTAYSY
TVKAKDAAGNVSAASSALSVTTLPTSGGSDTQAPTAPGNLTATGSTTSSVSLSWSAATDNVGVAQYEVYSNGALAQTVTG
LTATVTGLSADTSYTFTVKAKDAAGNVSPASAAVTARTQPTGGGGGQLPKHIVTGYWQNFVNGAKNLKLSEVPAEYDIIA
VSFAEMDRSKPGGVTFGIDPDLSKALGGYSNADLINDIKAKKAQGKKVILSIGGEKGNIDLSSASPNVSNFVSTMSSLIT
EYGFDGLDVDLEHGFNVPNLTTAVRQIQAKVGSGFILTMAPQTIDMQNNQTSYIQFYNNVKDITTVINTQFYNSGCMLGR
DGKCYSQGTVDFLTALTDLAIQWVQPSHLGIGVPATSSAAGGGYVSASVVNNALNCLATGNSCGSYKPVAKYPDIRGAMT
WSINWDGMNGYSFAKTVKPFLSQLP
>lcl|NZ_CP026520.1_prot_WP_241688784.1_756 [locus_tag=PC41400_RS03960] [protein=glycerophosphodiester phosphodiesterase family protein] [protein_id=WP_241688784.1] [location=883147..885198] [gbkey=CDS]
MQPAKFFKYLKPSTGKERKQSTRRWNIFYVGVTLALVGMAVPFRAGMPQAHMADAGSQALGSGTTIAGGTPDHPNILSDK
TKQTGAATAVASAGKGGTPPPVGAGAGIAAQPVKQPPAPGSKLVQVTEPETGIAIAPTIVTELRSKEQLEELSGPAKPAT
VILHVSSGLRITDPSGKTELGTLEAVLDGLGRRMIPAFYVRDGETADSVAGLLKKSGIEDAFVVSDRGELVKRARTLYPA
LRGIVDFSSAGNLSADDLLNIRRKTTASLAKIAILPGASASRSSVAYLQQRLIVVWTKEAGTKAGPELGLHTLITAGVNG
IVTDSPKAAFKALKVYSHGNTLIRKPYIIGHRGMPSKAPENTIVSNRLGLEAGADFIENDMYLTKDGRLVIIHDPLLEST
TNGKGKVEDFTLAQLKKLNANKPYPAGFPYVQIPTLEEQIDLARSQGKMIMAEMKSKNPAAMDAFVKVVKDKGAEDLINT
MAFNPDQLKRLSELMPEMPMGLLTGETANEANIDGSLTNTLRLMQLRNTTFNTTYKGLGPNFMEAAKHRGVLISPWTFND
KSNFAKFFCYGAFGITTDYAFWASDWAASVKPVQDHIVVGNYESKDLSVLMETFKGDRLEKAADVILLDGEISVEAYGSR
ITAKRPGKAHALLRYTAVIDGGSKYDLYTQPITLEVRDVKEPG
>lcl|NZ_CP026520.1_prot_WP_042232136.1_757 [locus_tag=PC41400_RS03965] [protein=hypothetical protein] [protein_id=WP_042232136.1] [location=complement(885360..885809)] [gbkey=CDS]
MSVVLKDKFSSNMDKANLVFAWQVVDNLTVEEAQEAARIITETVANNAKSGKVKLLVDNRFMERDRRPIVFTPEVNSIWE
EAQRTVFPHVSHIAVLCSGSIMKMQMDRISRNSNISDIQKSYWNKEDQVMLKEAFDFLGINSNDLVKNK
>lcl|NZ_CP026520.1_prot_WP_042232135.1_758 [locus_tag=PC41400_RS03970] [protein=iron-containing redox enzyme family protein] [protein_id=WP_042232135.1] [location=complement(885829..886470)] [gbkey=CDS]
MTSVTTTKEEVIERISGIISSELERVKTENPFFLNFPAEKNPQYFVWCNNLYHLSKHFGELLKLRWERFPDVDHDVFSTH
YDEEKDHAEMLRKWMIGIGLDDPENHTANYETEHFISLQYRAVAAMDENMSLLIINSTSEGFAHAVYLHAYKILQETGFT
DLEYWAIHCEADEEHSQVYHLIQNMTEKQLQEAEHLVKYTCNTLDVMLASWFK
>lcl|NZ_CP026520.1_prot_WP_042232132.1_759 [locus_tag=PC41400_RS03975] [protein=PLP-dependent aminotransferase family protein] [protein_id=WP_042232132.1] [location=886754..888133] [gbkey=CDS]
MREFTLDLSASKKPIFHQLYTYFKNGIREGRLRSGDYLPSLRSSARSFGVSKNSVENAYQLLISEGYLRNIPQRGFCVTY
EPDESSRKVIRAEEANTPEQTVTTDFRYGNIELGTFPFRHWNKVRNLLVAQYQSRYAVEGTSQGEYPLRKELSRLLYESR
GVLSTPDQLVVGAAPQQLVMLLSQLLDVKKHRIGVENPGYDGARNTFLNLGFRVHPIPLTGSGVSLRELTGSAANVMYVS
PSQQFRNRVTMPADKRTELADWARSREYLIEDDYEWEFKYEEGILPAIQSMVPDKTVYIGRISKALLPVFNLSYMVLPPG
LLSEFHSKVPEYDQPVSRLDQLTFARYLSDGYWYKHLQQMRRLYEEKRRYFSEALERYMPGTAQAEGKDTGLHAFLTVRT
DRSEDELIEAALEKGVRVYGTARYWLRPGDEHPTVLLGYGALGREEIAAGVKALAEAWF
>lcl|NZ_CP026520.1_prot_WP_042232131.1_760 [locus_tag=PC41400_RS03980] [protein=HAD-IIB family hydrolase] [protein_id=WP_042232131.1] [location=888476..889237] [gbkey=CDS]
MNFIFDLDGTICFKGQPISKRILDVLLELEDAGHFVGFASARPCRDMLPVLDDRFGGHLLIGANGAMTFYRKQPLAYQPI
PGPLAGQIVNLLNDYEAEYLIDDTWNYAYRLSPGHPFLANVDPHGLARRVELDDIQSFVKIVVLSCSRFEELSAKLRALD
VTIHHHSSEGILDVTSQGVNKMAALSGCGLQLDEFICFGNDNNDLPLFRKAKHSVLIGDHDALAELAKERIAVDDRTEEN
ILQKLKELGTVLV
>lcl|NZ_CP026520.1_prot_WP_042232129.1_761 [locus_tag=PC41400_RS03985] [protein=M24 family metallopeptidase] [protein_id=WP_042232129.1] [location=complement(889270..889962)] [gbkey=CDS]
MAASRQERHTNLREAEKKALALFELAEAKGILRSGVTEKELNREIYEFAFELYGIKKYWHKRIVRAGSNTLHPYRENPPD
RMVEPGDILFIDFGPIFEDWEADFGRTYVFGDDPLKNKLRRDVELAWHDGKAFFDSRPDITGSELYDFMTALARDYGWEY
GGPHAGHLIGEFPHEIVQGEEVENYIHPDNHIKMRDNDKNGLLRDWILEIHFVDREREIGGFFEQLLTVD
>lcl|NZ_CP026520.1_prot_WP_042232127.1_762 [locus_tag=PC41400_RS03990] [protein=hypothetical protein] [protein_id=WP_042232127.1] [location=890185..890544] [gbkey=CDS]
MKQRNAIIRWCLFLALSFLLSFSAPVPLTQEAAASQSSSTYESVPPLSEHKSKYVIRKAVTHHFAKMVKAANTLVVLFVL
CLFTLFKAPIPGLRYKPFICLLRRKQFLLPIKFTSNYVV
>lcl|NZ_CP026520.1_prot_WP_042232125.1_763 [locus_tag=PC41400_RS03995] [protein=cation:proton antiporter regulatory subunit] [protein_id=WP_042232125.1] [location=890875..891369] [gbkey=CDS]
MYVRESDLPGIGKKFLMNTRGGDKIVVILHDDGRREMYHFHYDDPDESLSMVTLDDDEARQLAAIVGGLTYKPKALESIE
VALDDLIIEWYKMDPEYACVGQSIGELDVRQNSGATIIAVVDKNHGKQINPGPDYVLSADSTLIVVGERQQQKLFRHILK
NGSG
>lcl|NZ_CP026520.1_prot_WP_042232122.1_764 [locus_tag=PC41400_RS04000] [protein=cation:proton antiporter] [protein_id=WP_042232122.1] [location=891374..892615] [gbkey=CDS]
MNHMVFEVGLAIALIAAAGLLSAKLRFSVIPFYILIGMAVGPHALEIWHFDFRFIESAPLIEFMGRIGVLFLLFYLGLEF
SVGRLIKSGRSIAVGGTIYILINFTLGVIFAWALGFPLAEILVIAGITTISSSAIVAKVLVDLKRTANPETEMILGIIMF
EDVFLAVYISILSGLVLSGSTTLGGVLLSALIALGFMLVLLVAGRKAVPLLNRLLNIRSNELFALVIFAALFLVAGFSET
IHVAEAIGALLVGLVLAETEHAKRIEHLIMPFRDFFGAIFFFSFGLTIDPLALGGAVWMSLGAVAITLIGNLAAGLLAGR
SAGLPSKASANIGLTIVSRGEFSIIMANLGKAGGLLAVLQPFAALYVLIMAILGPLLTKESKTIYKWMNKVIPFKDASGK
KEGKTVLQKEGSG
>lcl|NZ_CP026520.1_prot_WP_042232120.1_765 [locus_tag=PC41400_RS04005] [protein=YitT family protein] [protein_id=WP_042232120.1] [location=892698..893798] [gbkey=CDS]
MKLLQNVRMGPAGEVLGGWSLAQWCFKLLMVICGSVLAAVGLELFLIPNRMLVGGMTGLSALAAHFTEMRLGLFLFVLNL
PLILLSYRKVRRNFAVLTVLGLLVFSLSALILHPVPALIENPMMAAFCGGAALGLGIGLVIRYGATLDTLETRDHTEKLV
KFGFKDQDFMLLNCFILAGAGFIYGWDQAMYSILAYLLAYETVHFSLRGFSVHRRITIVSGHCAEIEKEMEARLGRKASD
CEAEEAEKREAAAKRAMPEERHAYGLERAAERFFSRRSGRVIANFRPVEPCESAGGGATGSIREPDERAPNEARSESFSR
SRGRSLSYTIHFMEEARMKAIVKAIDPAAGIVTDSRGRRRDGSLAE
>lcl|NZ_CP026520.1_prot_WP_042232117.1_766 [locus_tag=PC41400_RS04010] [protein=AraC family transcriptional regulator] [protein_id=WP_042232117.1] [location=complement(894046..894888)] [gbkey=CDS]
MNLMDTFTRPIEALYNTSRIPAGFHSHLYYELYYFHSGQCSYLVGHRIYNLSPGDLLLMNGMTLHYPKINPAFEYCRTIV
HFDPAYAEGLFRAPFHTPVLNPFKRTDNRLISLEPGERAILEPILEELCSLSGESDSLCRDRFLASFMRVLLQVYEIADK
PRNRSHEPAGPRQQHVQQLISYVEAHYTEDFRLDDIAAAVHLNKHYLVKMFKETTGITLFHYLYRRRINQAKVLFTLEPQ
KSITETAYEVGFKHLSHFSQTFKKFTGLTAEEYRKQVLNT
>lcl|NZ_CP026520.1_prot_WP_042232116.1_767 [locus_tag=PC41400_RS04015] [protein=SDR family oxidoreductase] [protein_id=WP_042232116.1] [location=895186..895953] [gbkey=CDS]
MNLFDLTGKTAVVIGGNSTLGGEMAAGLAGHGAQVAVVGRNPVTSAAVRDRIEQGGGEARCFQADATSSGDLRRVLDEVL
DWTGRADILLNCPGKNSATPFFQITEEEWDSIMSVNLKSVMLACQVFGKYMVESGLGGSIINISSVSADPPLSKVFTYSA
SKAGVNSVTQYLAREFAPAGVRVNAIVPGFFPAEQNRKILSEERISSIMSHTPMNRFGEARELQGAAVYLASSAASGFVT
GSILRVDGGFGAMTI
>lcl|NZ_CP026520.1_prot_WP_042232113.1_768 [locus_tag=PC41400_RS04020] [protein=glucuronate isomerase] [protein_id=WP_042232113.1] [location=896018..897283] [gbkey=CDS]
MKAQTIQNAAELRKIVERLAASTPVTDMHTHLYSVDFEDSLLWGIDELLTYHYLISESFRWSEDSYEAFWAMTKREQADA
IWKKLFVDHSPVSEAASGLITILDRLGLDVSSRNLEEYRRNLAGRDLRGQLDTVMKLAKVEHIVMTNDPFDAAERAVWLD
RGGNTDPRFSAALRVDALLNNWPQAVRELKQLGYRVEEAWTEETRGEVRRFLSEWIRRMDALYLAVSMPGDFVYPSDDHR
SRMIDEAMVPVCREHGIPFALMIGVRRSVNPGLRLAGDMSMHSDVTAVEALCRRYPDQKFLVTMLARENQHELAVLARKF
RNLMIFGCWWFLHVESMVEEMTRFRVELLGTSVIPQHSDCRVLEQLIYKWEHSRSIIGRVLADKYEKLHGSGWYVTEEEI
QRDFDDLYHRNFWRFLGREAR
>lcl|NZ_CP026520.1_prot_WP_042232111.1_769 [locus_tag=PC41400_RS04025] [protein=hypothetical protein] [protein_id=WP_042232111.1] [location=complement(897406..897810)] [gbkey=CDS]
MKRMILILILMLSITGCADQGGRKNLDSETLAKPLLNFFQGDITKITRIDIRNGATGEVKSFKNKETLQKWLGKFRDIEV
KSDPNQEPKTGYAYAVELYEGNQIKFSFTNSQIDKLYLLPSEELIKHMKSLYES
>lcl|NZ_CP026520.1_prot_WP_042232110.1_770 [locus_tag=PC41400_RS04030] [protein=hypothetical protein] [protein_id=WP_042232110.1] [location=complement(898002..899234)] [gbkey=CDS]
MLKKLTTKVLPVLLFATLIPSAVFASSNEVSSKDAVKAALFQVVLDQKENNYSKWKDLIVSASEPVLVYDISGEKHSYIV
NLKASGREVGFVEVSYYKDDNPILSFGYSSNRMADQEKSELRNKVKQMGKQVATEKVIINGPAKFALKKEFSDGSAEIIE
SKEVAELTKEQNQKIKKKERKENKEGKKFWKNIDDVTSEIGTTSDGVTDNPGAYESGYSSTNSKFWDGVGDANQYTNSLW
TGASGCAPTSGYNLMFYWHYQKGFSGLLKNSSGYVDKDDSVLLMRQKMGTDNAGNTSVSAKASGLRAYAVARGYKSSSST
LTTSPSWNTVKSHMSSAPSLLDFFDQSFYCGSSSCGHTVVGVGYTEYFYNGSSSGHQYMIVHDNWGNTPENVYVAYGRNY
TSMNSLLYSM
>lcl|NZ_CP026520.1_prot_WP_042232102.1_771 [locus_tag=PC41400_RS04035] [protein=sugar phosphate isomerase/epimerase] [protein_id=WP_042232102.1] [location=complement(901124..901984)] [gbkey=CDS]
MKLSVFTVATPELSPEQLAAAAKEAGISGIEWRYKEIPPEMAGETPSFWRNNLCSISPEWDDAELSRFRDAAERHGLRSL
SVTPYLDAGDLGGTERVLDAARRLGASFIRLGVHRYDRMRNFRELFELQRAYLLEAEALCRQYGVKGLVETHHVTIAPSA
SAAYRLVEGLQSEHIGVLYDPGNMVYEGFENYRMGMEILGPYLAHVHVKNGGWVSGADENGMTRWVSAARPMNRGMADWR
QVVADLRAVGYTGWLGVEDFSSELPGPEMLKKFSAFMNGLMQPSEA
>lcl|NZ_CP026520.1_prot_PC41400_RS04040_772 [locus_tag=PC41400_RS04040] [protein=Gfo/Idh/MocA family oxidoreductase] [pseudo=true] [partial=3'] [location=complement(<902573..903313)] [gbkey=CDS]
MCNGLREAGGELKWVYDPDPLKVKAFLAAYPSVRPATSLEEILQDEEVKLVAAAAIPCDRGPLGEKVMRSGKDYFTDKTP
FTAPDQLQSAKRTAAETGSKYAVYFSERLHVESAVFAGQLVRDGAIGRVVQVLGLGPHRLGAASRPGWFFERAKYGGILC
DIGSHQIEQFLFFAGCRDAQVLHSKVANYNNRQYPELEDFGDATLLGDNGATQYFRVDWLTPDGLSVWGDGRTVLLGTDG
YIELRKY
>lcl|NZ_CP026520.1_prot_WP_042232098.1_773 [locus_tag=PC41400_RS04045] [protein=AraC family transcriptional regulator] [protein_id=WP_042232098.1] [location=complement(903580..904437)] [gbkey=CDS]
MFTNLHDRKDSLGLSYRNEGKVDSVFHSHAYYEIYYFHSGSCNYLIGGNIYMLQPGDLILMNGMTLHCPKVDTAVPYIRT
TVHFEPAALKPYLELPHAVNVLQPFQELKNCRLTLRGPEKEEAERILHEMSIHGKTDGLIAYNRLHLAFLDLLYFVHEQC
RRTLRAGREFSSEKEKHVQRIVSFLEERFADDLHLEHLQKHLNFSKFYLSKIFKEITGLTIFQFVYRRRINEAKTLFLHN
PGHSVTDVCLQVGFKHLAHFSRLFKQQVGLTPEKYKKTVREQLSS
>lcl|NZ_CP026520.1_prot_WP_042232095.1_774 [locus_tag=PC41400_RS04050] [protein=Gfo/Idh/MocA family oxidoreductase] [protein_id=WP_042232095.1] [location=complement(904747..905907)] [gbkey=CDS]
MEVVRIGIIGLGNMGSAHAKYLIENKVKGAVLAAVCDARTERLEWAAREFGNGVARFAGIEEMYASGTIDGVIICTPHYG
HPAEAVAAFKHGLHVLTEKPAGVYTKQVREMNEAAAAAGKVFGIMYNQRTQPLYKKLRDLIRQGELGEIRRINWIITLWY
RSQSYYDSGGWRATWAGEGGGVLINQSPHQLDLWQWVTGMMPKRLRAFCAFGKHRDIEVENEVTAYAEYENGATGVFITS
ACETPGTNRLEVVGDLGRMVIENDKLTFYRLRQSETEFNRTYKGGFGEPEVWKIDIPVEGQRGQHHLITQDWVNGILHGT
PLIAPGEEGLKGLTLSNAMLLSAWTDEWAELPLDEDRFYGLLRQKIEASGAAKTTEEKTLDTTGSY
>lcl|NZ_CP026520.1_prot_WP_042232093.1_775 [locus_tag=PC41400_RS04055] [protein=extracellular solute-binding protein] [protein_id=WP_042232093.1] [location=906148..907491] [gbkey=CDS]
MGRKTARISAAALAGVLAVGLAACGRADQSAGTGNTQDTNTGSKGGDKKVTITFQNIYPDPTTPTYKMLHKIVEQYEKEN
PNIHIELDSMNTDQQKMKLKTQAASKEIPDITIVNPAAQMKPFVDAGLLAPLNDMLDKNGLKGTFQDGLLDYYSFNNSVY
ALPDGNNIQVVYYNKDLFAQAGIKDTPKTFEELLEDVKLLKSKGITPIAIGEKDSWTGSFLFMNMLLRTNGGPGFLQDVL
DKKKTFNDPAFTEAVSKFQELVQAGAFNEGATSVDYNAGGNIFKTGKAAMHIIGTWETGAIDASSVGGKVGVFQFPTVGG
KGDVNQYMIAPGSAFAISANSKHLQETKDFLNYFTLNLPKVQFELKNAVGIGQKVEGDFKAAGYSDLAISILDLFKNVKG
GDLAFDNTMNPATAQVHLGSIQNLFIQKADPKQVAQEHQSAFEANIK
>lcl|NZ_CP026520.1_prot_WP_042232091.1_776 [locus_tag=PC41400_RS04060] [protein=sugar ABC transporter permease] [protein_id=WP_042232091.1] [location=907612..908502] [gbkey=CDS]
MNVLKVPARTIAVFVLPSLLLYVCMVFVPILVSFYTGLLDWNGISGSKFVGLANFKTMFFDDPVFWPSVRRTLLYAVFSM
IEIPICLMVAILLNRYVKKANSLVSIYFLPVILSVVIIGQLWKTVYNPASMGGMINGMLTASGLTEWTHSWLTEPKIAMF
CLYIVSLWQYFGYHLLIQFTGVQNIPAEIYEAARIDGAEGLTADRYITFPMIVPILKISLVLAFIGSLQAFDLVMVMTGG
GPAHSTDVISTHMYNMSFLSQKYGYGSAIATFLVILCLAFTVLINVVFKRLENKVS
>lcl|NZ_CP026520.1_prot_WP_042232088.1_777 [locus_tag=PC41400_RS04065] [protein=carbohydrate ABC transporter permease] [protein_id=WP_042232088.1] [location=908517..909341] [gbkey=CDS]
MKSVKKGIVYTLFAILIVTQMYPLLWLLTYSLKTNEEILGSSFFSLPASPRWGNYKEAYEAGHYLQYLFNSVFVTVLTMV
CVILLSSMAAYAISRFRWKLAPAVMLIFLIGLMIPLQATLLPLMIIFKNMNILNTHLSLILPYIAFQTPVAVFILSGFMK
SVPHEIEESAYMDGASVYRIFRSIILPISVPPVMTVCILTFINIWNEYILAATFISSEKLKTLPFGVYTFVSQYSVNYGN
IGAFLVMGAVPVILLYFILSEKITKGMVAGAVKG
>lcl|NZ_CP026520.1_prot_WP_042232086.1_778 [locus_tag=PC41400_RS04070] [protein=histidine kinase] [protein_id=WP_042232086.1] [location=909395..911707] [gbkey=CDS]
MRRGFQSIHHRLFFLFLFCMSGILIIISLLFYSRTTEQFQTKISDLSRKNVSQTVDLFDLLLKGYDSLSKTIGNNFDLVR
LLTESPQEPALDYINTRSVTNMIGAIFYSREDLIGIHVIMDKGKIYNYGNYMNVVDPEYRRADWYRDVRQSSGKMVWLGV
FPHSLIDQNEKRTVFAFGRQIYDLNEHKPIGIVLYEANPQAILSALNNLKLSEHSEVYLVSDKGRFVSSTASAEALPSLQ
DIPLPDDSQEMTVDQKNGRLAVASKLPFADWLVVSLTPYKDLNVELVQMQRYIILIGTILVLVSAFIASIVSKRISMPLI
RLIRQMKQVEQGNFKGIVNVTSYREINILVGSFNHMVNRIEELIERVKISSVSEKNAELHALQSQVNPHFLYNTLDMIYW
MMDEKGNDKLGEVVLSLSHMFRYSSQWEEGAEVTLREEFGQIHHYLTIILVRLEGRLEVELKTDERYMDIRIPKMTLQPI
IENAVKHGLEPLLNRPGMLTISTVADSRELAIRIEDNGVGIEEEKLRRLRESLYGLCTAGGRRSGGGAPDAAHGPAGPGA
GGGAPRTDAAVAAGEPGGREVAGTGTDGAGGRGPTAPVAGAVGAAVFGTVGEALEASASTGAGGAATNAAAGGPGDGAAC
GAFPSGAGVSGTDAAASCAVAVPSSGASASSGTDAGSASSTVAAAGPGPAQAPTHAGTPALGADPWFPSRPVAGGSEEEA
SSRGGIGLQNLHRRVQHMYGEAYGVTVESSAGEGTTVIVRVPLPKEESNV
>lcl|NZ_CP026520.1_prot_WP_042232083.1_779 [locus_tag=PC41400_RS04075] [protein=response regulator] [protein_id=WP_042232083.1] [location=911704..912882] [gbkey=CDS]
MNILVADDESIIREGVKRTIENAYPGYGVFLAASTEEAVKILGGQHIDIVLTDIIMPGMTGLEFMRISKRKHPAIKWVVI
SAHSEFAYAQEAVRLGARDYLLKPIGKPKLIELIESLTGEIRRENLQSKESVKLKSSLKYLREAVFLRLAAGLDTGNLDI
QSFVEDFPDYYLVFVQMDAGNKSVHLEHFIVENVLSEMIETHGQGFVVSYDRQSLLGLVALSEGIRIETLLDALKSHLKH
YLKVPFQILNSGLHHDFKSVPQTVTRLRQASAAVEYDSIKVSGDKSIEVALQYIKAHYSEDLSLERMASVVFLNPVYFSQ
LFKQKTGQGYKEYVTGLRLDQAKQLLCNPLLKLAEIAERVGYQDVRHFTQVFRKKFMQTPTEYRQEHNVTVV
>lcl|NZ_CP026520.1_prot_WP_042232080.1_780 [locus_tag=PC41400_RS04080] [protein=serine/threonine-protein kinase] [protein_id=WP_042232080.1] [location=913095..914087] [gbkey=CDS]
MSVRIEYGLAAKTFLQGGAYRIGRVLACSELSVVYAARETGTGAACVVKEFFPKSLALRDVDGKSVLCRTPSLRSKYETL
KSVFLNEAVILEHLRHPNIVRLTGHMEENGTAYLILERCRGMSLERCIRKNRLRSKSRFYATTVPGLLDALEYIHEQGVL
HRDVKPSNIIVGPDGQAKLLDFGSAVRSQSEDKPIFTSAGYSPLEFYSAASRQGPFSDLYSTGATLYYCLCGEAPVEASR
RLFHDPVEQGLRKRGGRATPLLSRRIMNCLSMDAGQRGSLQRLRTAIKAESLLWKAVEGLRANFRPGRKNSSPTNEAPSG
KKKDLVKSDV
>lcl|NZ_CP026520.1_prot_WP_042232078.1_781 [locus_tag=PC41400_RS04085] [protein=VWA domain-containing protein] [protein_id=WP_042232078.1] [location=complement(914180..916234)] [gbkey=CDS]
MSLVRKLIHSKNWLQAALLVSLVAGLLIYPRPLSAAKVETSQGLDAMFVLDTSFSMNETDKDGIAAEVVKMFMDLEGTGQ
SRVGYVAYNDKIVSSLPLSDLGSSDRSNAWKSTLDGIKRRGYSDMGLGLRKAAELMETGYDPARKPFLILLSDGGTDFGK
HAGGRNADISGRDVEDSIGIARKLGIPIYTVGLNRDGSARKSELENIAAETGGTSFMAGSADDLPEIFNLIFAKQLQSSL
VSVSAVSATGELQEVTVPVANSSISEANLILLSPNPLTEAQVYSTSKNVRLYKSDNYTLLKVIEPQKGSMRVIFRGKPGD
LVKVNLLAGYKLQAGVELPEEAQKGVASPFTARLYHPDGKPLGDSDLYGSMKTELVVTNPQKKTEERFPLKQEGNGFELA
HIFPATGTYPWKIVMTAPDFYRETKPAELKVLNTPPEAKPGTIKISREDGEQKIDLGEYFHDANGDKLTYSVTADGGRNS
VEAKFQDGFLLLNPLNTGDSELRVTATDEEGASAMAILAVTVVSIWTLITKILLGVTAAAALGAILYFTLRPKPKFSGRL
EGYFLQTASGNDIPVKYWPLTSFDTRSISLQDLFRSLNVNESLPEADRIFFKARKDGSLLFRHSTRCAVQKGASPLGKKQ
KDKLEYGDKLYITFEDGVTEIELRYKAVKASYQPQTQEAQTVRT
>lcl|NZ_CP026520.1_prot_WP_042232075.1_782 [locus_tag=PC41400_RS04090] [protein=hypothetical protein] [protein_id=WP_042232075.1] [location=916663..918831] [gbkey=CDS]
MDIINQTNRTMLFEEINPEKLDLVTIVGDVKAIDSLSDDKIKEINRHLLVRNFDEFLDKFSPTVYSFYNAANQKVMYTLK
KPEGLADDAISEIRIDQNNDFLKMLFTLIDTKRSQGIPNVDFKFENLLDMISPKKVMDDIRQVRKEIHYLFGQYEKLDEG
DPKKFDLGDKLNGMFEDASKNYNNVMAMLPLAIEDIKTRLLLGSAQEENKSEAFQIGMLTIGNNGELKIIEAPKSGSTEL
MLLDDNSSNELVSVFEEDYESITETPSGYVKDLVVRTFCPLPAVQSTVDVETEVQNYNTYLEFYKNAKDDFVKTVKPLVE
KILGVKMFFEQYATKNKGMQPTLLVTNSKLDMTVKSSNIPRLETYLNTVNSKNDFSDTIWFGIVPSVELETAGKVKVSRE
RFKGNEKVMKPDGNTMESLSMLLQVLKEYKVQLFFSFESGEDTTFNSMATAGIDKYIDKCGVLVRKEYSEFAVPCVPNFT
IIPKDKSGVVIDSRMLHTEDGARLSQEKEDILKLWIEGVYIGASYVAAGLVSAYQCPEYLREQFKSVSRDYPGVRFDIEA
GDHSLRAVSTMAKEISGFTNNIKDAINRKNFGFIFSSENAQLQDRDIKRITVYKARSLAMAEDGFDSIYKTLVSTYIERI
LRFQTGDFKHENIVKFFSNNPSSQKSKWLGTRGFVNSIIQDGDDMSYIIDDKSNLCHIDLVFNGNVKNLEVTITKGTSPV
KV
>lcl|NZ_CP026520.1_prot_WP_042232073.1_783 [locus_tag=PC41400_RS04095] [protein=hypothetical protein] [protein_id=WP_042232073.1] [location=918951..919376] [gbkey=CDS]
MGFRLKVEGAETFELGMDNIQTVTYDTDTPNDSNARSTDVGATLRISGKIITATDGDSADDTLKLGLWSLVPAEKADCYR
KVTLEVIAADQVVRKIYFPNAFVVDYSENFGDTEGVGSFSLFIKQKKDKTEFAKVEGGYSI
>lcl|NZ_CP026520.1_prot_WP_042232371.1_784 [locus_tag=PC41400_RS04100] [protein=DnaJ domain-containing protein] [protein_id=WP_042232371.1] [location=919566..919997] [gbkey=CDS]
MDYYQVLGVARNAPQAEIKKAYRLLAKKHHPDLNPGNPDAERKFKQIYEAYQTLEDEASRNAYDHKLGGQNRRESGGKGP
HEKAAGSRAGAQQAEFDPSDVRKNFERFFGFAGKAGGGEAKPSQKNPMDTTDLFNRFFDAKKK
>lcl|NZ_CP026520.1_prot_WP_042232070.1_785 [locus_tag=PC41400_RS04105] [protein=FHA domain-containing protein] [protein_id=WP_042232070.1] [location=920192..920743] [gbkey=CDS]
MGETGVKGRKSGWITLLDVLIAGLAAAVLIYVYFINIDNTLKMFVSLLMALGLSVLGFVYYRMPAEKPEREAIAKLVLLN
EDGESVKEWYIQGETSLLIGKSSGQYEVDIDLADTEYASLIQSEHAVLNRSGREWFIEDVDSESGVGIQKSGRGPRDKQE
VEEPHKLDVGDIVYIANTRLLVK
>lcl|NZ_CP026520.1_prot_WP_042232067.1_786 [locus_tag=PC41400_RS04110] [protein=FHA domain-containing protein] [protein_id=WP_042232067.1] [location=920804..921340] [gbkey=CDS]
MSLTRCANGHMFSTRKHGNTCPYCSIVVEPPASRAEESRAQRVQEDDKTMPFLGETTGIEPVAGWLVCIEGPQMGQDYRI
MAEKNFIGRSEEMHIRIIGDNTISRRNHAVIVYDPKKRNFYLLPGDASGLAYHNNEAVYSPVELTAYDLLQLGRSKFVFI
PLCGAHFEWNNTAAAGIG
>lcl|NZ_CP026520.1_prot_WP_042232064.1_787 [locus_tag=PC41400_RS04115] [protein=protein phosphatase 2C domain-containing protein] [protein_id=WP_042232064.1] [location=921340..922182] [gbkey=CDS]
MPAGSDGTPYLIVGAAILLILLMLYVRSRLNKRKPEVPGVRIGNGQTIGARNEQDDYFASTTTSIGTLAVLADGISGLAN
GRMCSTLAVSTFLKNFLNVEHVRHIPYFLSETARRVNSEIVEQLRGAQGGTTLVAGILNGDHLYWGAVGDSVILIYRQGE
FLPVNPKHTYGNVLEQKYLSGEITKDEMEANPKRRQLVNYLGYEQFQNMEISETPFRLKTGDKIIFCSDGVYNTLTEVEM
EQILTGSGSPEESAQGMIDAIEAKGLHKQDNATVIILEKG
>lcl|NZ_CP026520.1_prot_WP_042232061.1_788 [locus_tag=PC41400_RS04120] [protein=serine/threonine protein phosphatase] [protein_id=WP_042232061.1] [location=922233..923996] [gbkey=CDS]
MRKENSDFKTGFVTEAGTFVQNRDYFAFVELEDMACWVLADGLDSDSEVNSAEMVVRSILSRFLNKPTLSAARLKRYMQE
ASEWLNEESRRVRLKASVLIVVTDYTRIVWAVAGSARLYVIRGGRLAERSRDQSLAQSWADEERIPLEAVDRHEERHNLL
NYAGKPDVFEPYVSRKMMLEDGDVLLMSTPGMWENVASAEMVDAVEEAKGPAELADLLEEVLLSRQKRMVPNYTAAAVYA
NKVYKEVKGKKRKWAKRIALILIPLLLVGGGAYVYAAKEAARKAEAAQGMLEHGKSGDDYAAEGDYAKALKEYSEARSDS
QKIKDKVHTVLFTKKQRAAQYIVDGDTFLKEGEYDKAAASFTKAQSEIKNQDDFNPQHLEERLGRVQAYSAVMNTVKEGD
QKFTALDYIGALEIYRKAKQASAEASFTDGDKLIKGKLDETQEKIAGLEKETKLLEADKLDKKGDRSMAAQDYAGAITAF
ASAQEIYQEIKMLERVLGMERKITKAEEKQNPVPSAPPPAPAAQGGQALPGQGSGEHPSGSNQPAQPQTPPSAGGQSSPS
SAGASGSPAGGEGQARPAKGEGGDSTA
>lcl|NZ_CP026520.1_prot_WP_042232059.1_789 [locus_tag=PC41400_RS04125] [protein=hypothetical protein] [protein_id=WP_042232059.1] [location=923993..925279] [gbkey=CDS]
MKDIILDRLNKMEDLEQRRMLKQLMTGLFLNLVDYQEQMNRQLEKRVFDEIEDWEGRFDIYASLCSRDEWDPLHEFLYPM
CPEDVENTLIDMLHVIEGLRDGREERLISLFLECDSVRIRELTKSGRLFDGHLKTLSGTYPIKVALRPCEKYIKEIENLY
QIFLKNGIPWRTINHPFAYKFIDVVLTECAAIPGEDETVTEISVNLEEFEAYKRTDRIPLWNMERLEIKNSGFPVPASDK
VNFEHVLSLRKTGMEHGYLVDGHTDDIRYIKRSGEELTIVSPQEKAGVWNVLKVVQPVDIQIGRLQYALVSNRCASSFVG
KYARKQATIVRSKGEVARIVHSLEASEYLTLEDIRIVDKADSAAVTYEMNPFISDQVRASSGRAALCLSFRPGRSFGRDE
FIRNDFMSFVVSEVQMYLPEYTCEGVWV
>lcl|NZ_CP026520.1_prot_WP_174446831.1_790 [locus_tag=PC41400_RS04130] [protein=iron-dependent peroxidase] [protein_id=WP_174446831.1] [location=925270..925965] [gbkey=CDS]
MGLNYIWDLLIRAEQCGLDKKDIQFSAASVYSPYMELSLEHMNASHVEKQVEINPFYRFDEMFRGMLDINYTEYPEFRGA
LFDILIHFLGEIDLLQGMNKREFYIRFVRRDIEAGIFGDKVRRNFQKCGKEEQDIVAASVLQTVQTGDALFTLKRTARLL
FPGSTIYANCEEKNELLFYINQEESESGRAKISLLLDLLLPVRFHTELYWNRHFGILGVEETMRLDRIAMY
>lcl|NZ_CP026520.1_prot_WP_042232055.1_791 [locus_tag=PC41400_RS04135] [protein=molecular chaperone] [protein_id=WP_042232055.1] [location=925988..928639] [gbkey=CDS]
MNGYSYRLYQHPEDRMADPGVVRYSREELTEMTTYQLRTICSKEKLIEGFANTLDREGFIATILKFRGTEERLLIGGPVP
GGEERLQTAIRRYLRTPLGERGTKVPARITLYEGLRLDERDGCRLEGAAAFSGTNALLVNEQMELCGILNVVESVSKPGT
FCLTRNRDAYLKRSANRNYSLLFFSRNDSDYLYRIYHQNKPLPPVNLHYCRLPAADLEIREPELTEAVLAIDFGTSNTTA
GAYLDGAYVHTPCSQDLLNGTVRLDAINYAAFPDPAAPNGQWIELLPTVVRVADCSDPSHIRYQFGHEALLYRKKLGSGT
QATVFHGIKRWVNDYARSEEVSDASGNSTHVRRSDIIRAYLAYIIGTAEHQFKCRFRNLHMSSPVKLKAQFLDMYAEILP
EYRLETEDSLDEGMAVLYNTIADRFQQNRFEDGETYKALVIDCGGGTTDLSSCRFRIEDGHMSYRLDVQTTYENGDTNFG
GNNLTYRIMQYMKIVFAEYYSCGETVTDIDSLIGIPSGDVFRHVDEFGVASVYERLEERYREAELLLPTRFREYGSHTRE
DYNRVRNNSHFLWEMAENMKTEFFRKTGILRNRFHSGSGDVREHDLKITGVDRWYLSVREQGRFKDRYEFPDVVFTIQEI
NHLIKADIYEIVRKFLEDFYQSGELQTYSIIKLTGQSCRIDVFREALKEFVPGRTIQFTQKAGETDRVPDLKLACLRGAI
RYLGDKKAGIIQTHVTHQAAVIPYAVTAYTHSRSEKVLISTLDRVNTAQGSISRPWGAAEIEFCLKDMNGKLRQKYSYPS
RPDSFQPVLYEDIAETYGRNIPQDETDSIVNEEVKFFVFGGGQNWGFHVVPVARRREQLYLGAKRFFAFENDLSEIDFFD
GLK
>lcl|NZ_CP026520.1_prot_WP_042232052.1_792 [locus_tag=PC41400_RS04140] [protein=hypothetical protein] [protein_id=WP_042232052.1] [location=928666..929442] [gbkey=CDS]
MFTRTYPNFAKGRILKTEMLENLRDYPHELTDLFLRDYSDGIVAGTEIRVGEDSLCIGPGIVKHGGMIYTLTREVELPYA
ATGRETVLKIRFYPAETEGDFTVCRAEPVLDEHVSVSSEELELARFKLKEGARLRSDYRNFADLATEYNTLNRLHAQYAG
FRQSTLHPEIMRYFAAELMKSRASGTQDFTFAMQCLSVQPVDREAILHYLALRLGKEYKTYTNVQIHTHLTRVLEEAGSG
GRPVTGGRFGAPQRMIVD
>lcl|NZ_CP026520.1_prot_WP_042232050.1_793 [locus_tag=PC41400_RS04145] [protein=hypothetical protein] [protein_id=WP_042232050.1] [location=929472..930071] [gbkey=CDS]
MAFSDEKIIAMLNKMEESQKTNTIEEGPVRIGQRFYEFEQQAFYDDKLLVFIPKDFTDMPEALSAIKYPHSQRPETIKTN
EDGSINITLNRIDQDLEEGWVEELTQGMKTILKKVNPSNVFFTDGVEEIHGKQVGYFEFKSPALDAFLYNLMFFLELGGK
TLMGTFSCKYDEYEAWRDVAFQIVRAVRVVQVEEGAEPA
>lcl|NZ_CP026520.1_prot_WP_042232047.1_794 [locus_tag=PC41400_RS04150] [protein=contractile injection system protein, VgrG/Pvc8 family] [protein_id=WP_042232047.1] [location=930068..931480] [gbkey=CDS]
MSQTVFSFADISVAPYKFERIMDLHLDKAVNEHAKLTISGIVPEEMLDRYVEQADENEQIAVSVRDGERTTVIFQGIVTN
MAVKAVHNVRTLTIEAQSSTLLMDIRKVTRSFQNKQMTYRGLLDRIAGSYPNSDVVVEAAGGTSIGGLVVQYKETDWEFA
RRLASRLHLPLIPLCSQKGMKCYVGVPDLGEPHKVDAYNYSIRKNLKDFKRKSENGVSGIDEQNSISYEVSSSSILELGS
PVAFQQRKLYVYRASTRMEGGTLTSRYELRDKQGFSCPTLYAYKLAGTSLFGSIKDVSKDKVKVKLHIDGKSSSDESLWF
PYSTVYSSPDGSGWYCMPEVGDEVRLYFPDEQEKNAYAASSVDTDSSDPQKRSDPSVKSISTKYGKQIVFKPGSVEIIGS
GSLLMRLTDDGGIEINSHKKISISAMEDIEITGGGKVLIQGEEGIDLKQADASLSIMDEVKISGAKVNIE
>lcl|NZ_CP026520.1_prot_WP_042232044.1_795 [locus_tag=PC41400_RS04155] [protein=hypothetical protein] [protein_id=WP_042232044.1] [location=931477..932112] [gbkey=CDS]
MSRRMDKEAALEDFLEKHIFGKWQEDLMTVDAVYQEHRQRIEEEFLAAFDAVCGQAAELQAKGCKGAIRYIYFSLMRTSV
MENSPCYRIEAYDENWLTDTADCRSEWRADFIFAPLFRRMEELEETKKSYVRRVTSADLDRIKQIEAVKYHLLTVEFMRN
MVPQLLVCKNYQQMDKFPEIALLAGEYRDECQLLHGSLESKSGASGEALSG
>lcl|NZ_CP026520.1_prot_WP_042232041.1_796 [locus_tag=PC41400_RS04160] [protein=hypothetical protein] [protein_id=WP_042232041.1] [location=932109..932672] [gbkey=CDS]
MMDYFVLKQDERCTDAPVLLDVRSRIDLRDIRPDRAHRIADTVVMQVKSTPETEYPDVLDGQLYLISDRLKRIAAAYEPD
LPVKLIPLMDRVQRKQSNYYLPLFEEAEVLSPSSELNRDGTVVKKLVLQRSKLAGRKIFKIKESAKSLIIVRLDVAESLL
RRDFAGIRLERAAVEEDTKLEITENYS
>lcl|NZ_CP026520.1_prot_WP_241688785.1_797 [locus_tag=PC41400_RS04165] [protein=hypothetical protein] [protein_id=WP_241688785.1] [location=932797..933228] [gbkey=CDS]
MFRWLVSLGFLCAVSALLLMPSMEMDAEESGQFIAIIGACAFILGFLAIFLLLSKTVVTDHEVRSHSWFASKTISFSDIE
KVGYSRFFGGRFILQGTSQAVKVPIMSVGSAEFVDKLSEKLGRERAGAAVEALAKQKQQIERL
>lcl|NZ_CP026520.1_prot_WP_042232037.1_798 [locus_tag=PC41400_RS04170] [protein=DUF4280 domain-containing protein] [protein_id=WP_042232037.1] [location=933257..933646] [gbkey=CDS]
MTGPRYVVRGAYIQCQCGTHKRHINLPNSHGSYVNEKPMMNEEDYKPDNISHFGICISPLNQSGETIYLISESGETISGK
PCLPSILAKWTYSKETTKVAGKPALTTDSELHCELQGVIKFLNDGQHDN
>lcl|NZ_CP026520.1_prot_WP_084706651.1_799 [locus_tag=PC41400_RS04175] [protein=restriction endonuclease fold toxin] [protein_id=WP_084706651.1] [location=933669..936356] [gbkey=CDS]
MIGYDQIRVTSPYRMKRIDHVEITWKPGEHGRMKLRGIVDDADHVGAVLEASLKDEIQVFENGEDGEEPIFKGLITDVQA
VFTQGVYTLEIEALAGSYLLDVRRKRRSFQQADMDYPALIAAILADYPGHDVICEAGAGVPIGAPVIQYDETDWELLKRL
ASRFHTVLISDIWSVQPKFSFGIPKLKSRTLPSDTSYKAFKDLLAYQKAGGSGAGLHDTDFFGYEFESGIRYAIGEELRF
KDKPMRVSEASGRMEAGQLKYTYRLSRLEGMYQPSLHNDKLTGISLEGKVLDVKGELVKLHLTIDKEQPKTSAHWFAFAP
PTGSAMYCMPKAGTDASLYLPDDSCSNAKVLDCVRKNGDSCAKTGNPSNRYFGTEHGSEVELTPTAVNVAGGSKEPLKLS
FDDASGVTITSHKKLTLNAGEDISLYTPKRIVIKATSQLMAKKLAAQSGFAIEGEYHFLSENVLAEGRDQTSYPSYNDEP
QKGTPPPPPPPPPEKKFSWGGLLIAAVAVVAVVAVVALSVATFGAGAVLAGAAIGAVIGAVGAVASTAASDIAKGEFSGV
GDYLFNALKGATVGAVCGAIFGPFGGLGGAGGSLLPQTGGQVAQKLGAMLFIGGVDNYTGYVASELMEGREPTLAGGMGA
FKDGVLFGGIFAAAGPTVKWVKDGALSVGTKLYLKMDDLAQAASNLGKKYLQEAKSYIDGLSGISFGQRQPALVGIGSTP
NIQLSKAEQGPKPYYSDTHENYKAYMKETQDKAAKEATAVKSEGTGSVEDKLKLSDKARELMQKGHYSQAMDRHYEDLVR
KKTGGKEVIYGEKGRERELDSVTKDAVIETKRSMSAVTKPDNFINKKMKQQLKATIQYAEEHGLRVEYWFKYGVHPKIRK
YLEEKGVIVKTGMGD
>lcl|NZ_CP026520.1_prot_WP_042232034.1_800 [locus_tag=PC41400_RS04185] [protein=hypothetical protein] [protein_id=WP_042232034.1] [location=936358..936735] [gbkey=CDS]
MGKYISIRGWFECEEHDVSKVKEICREFTENYSDTELNYNTRKLYQSGWVFPENQVNWTTYIFYGADIREYHLDFIKEQI
SKIAVIPEITGYFLIDDHEGEKKICWQVFDGHFVETEQINILFNN
>lcl|NZ_CP026520.1_prot_WP_241688786.1_801 [locus_tag=PC41400_RS30180] [protein=hypothetical protein] [protein_id=WP_241688786.1] [location=937060..937719] [gbkey=CDS]
MPKAGTDASLYLPDDSYSNAKVLDCVRKNGDSCAKTGNPSNRYFGTEHGSEVELTPTAVNVAGGSKEPLKLSFDDASGVT
ITSHKKLTLNAGEDISLYTPKRIVIKATSQLMAKKLAAQSGFAIEGEYHFLSENVLAEGRDQTAYPAYNDEPQKETPPPP
PPPEQRQWRLRPSRISRAGKSVTLEIMRRMLFENPLSEPYPEPYSDHWAQAPRYWARWL
>lcl|NZ_CP026520.1_prot_WP_241688787.1_802 [locus_tag=PC41400_RS30185] [protein=hypothetical protein] [protein_id=WP_241688787.1] [location=937680..938966] [gbkey=CDS]
MGAGAPILGKMVVGGVSGAAESIVSQLTDGDPGFSWGQVLFTAGVGFGTAGLLDAKIAQKTIGEYISQGATKIAPKWIAD
GFGKIGEGFGKFVGNLNDWGTSLGKGYMKETQQFAQSIASSVTGTLNKSVNRLTKDIADLGQMGKELGQKFKNSVQMPRF
GYELAFDGPPLLANQIGNQFGKSTTNSANVLQSSSKAVTRAVGGGGTGGTGKTSGEGLPLDRKKYYRQKIDEANARGDLK
AADDARYERYCEEKLDKGEKPVSRKKWDELTEQLRKNRANGIEQEIAGRNALEKQLGRELKDNNTGEVLTFESSEGHLTR
PDSIARNEKGEIDLIHDHKHKSSSEDQVVYNDSQMRAEREMLEAGNGKHIVTISSDSPNLDAIPPRPRPSAPLGKESTVY
YVDPESGKITRIWKVDESIQGGGYWVDP
>lcl|NZ_CP026520.1_prot_WP_042232033.1_803 [locus_tag=PC41400_RS04200] [protein=hypothetical protein] [protein_id=WP_042232033.1] [location=939002..939778] [gbkey=CDS]
MTPNLFLIEMYPTVSFSSEEVLGRILDVFESDEKLVPTHWGNDERIKVEYNRNEIIEKVKEGRITEIHLHRDKSIKYSGY
FHVNLSNRSFLYFNFHKSLSKKHWSMVFEISDKISNIAKPRYGITNIFWPTTYPWNTENERNQIWMNLCSQPVPVEFLQN
GPLGVGSRTYFSNDILELFGREYLKNSPGHVSELDWEGICMDVIEKPWEADSSELLNKWLQVMKYLETSGAIAVPKFSNN
RMGATFSPSAAWQNYLNR
>lcl|NZ_CP026520.1_prot_WP_042232031.1_804 [locus_tag=PC41400_RS04205] [protein=pentapeptide repeat-containing protein] [protein_id=WP_042232031.1] [location=939802..940407] [gbkey=CDS]
MTGQQDRSNILKILDSYKINKEVIYGLEMSGVDLAYTDLSEVMGDCIQAVRVNLRASSLIRASFLDCDFEETDFSNSNLE
KIDMGDCVLNGANLYKAILKGAKINISMCRGTCFDEVDFSGGEVVGSSFSNASFKNANLSRINAQQASFSNADLTGATLT
EGNFEEADFRNTMLDGVVWTGALIDGAIFDEGVMEIVKSKL
>lcl|NZ_CP026520.1_prot_WP_042232029.1_805 [locus_tag=PC41400_RS04210] [protein=SMI1/KNR4 family protein] [protein_id=WP_042232029.1] [location=940684..941490] [gbkey=CDS]
MQMPIKSNHIPPIGDCTNLFERLSKYISRFDEKWVNEIEPAKKEDIDTLKNLTQINKYNYHFPKEYEIYLNYMGQDDKGL
LKTQLPGYASISEIINSYEGIHEEAPDTLSDKYIHFFQTESFYGQLSFDFTQTDNPQIVMTDEDSQFVSYFADNFEKFLF
QCAFSKYERLNYDKCIIFAGTPNMLKEAIKKHNESDVFGIIDKFSKTYDFQRAWFSDLTHHIGFKDGISFYIDSRNNSLR
GFVAGDLDKQIDNIGETLLAELYVNKNN
>lcl|NZ_CP026520.1_prot_WP_042232027.1_806 [locus_tag=PC41400_RS04215] [protein=hypothetical protein] [protein_id=WP_042232027.1] [location=941621..942163] [gbkey=CDS]
MEHNLVQKLSHKMKIRMWASCLKPAKINRNGEMVISPHWFIKWGVFLLALVFVAVTVVLLFDPKIKEEAVAACGMFALIL
GFCSIGAAISKMVINEETIRTYKWYGKKTIRFADITRVESTTLYGGAIVIRSNKKSITVTLENRGIHDFVKHLSRKLGSE
RVAHASAIISERIRVINSYQ
>lcl|NZ_CP026520.1_prot_WP_042232025.1_807 [locus_tag=PC41400_RS04220] [protein=hypothetical protein] [protein_id=WP_042232025.1] [location=942398..943054] [gbkey=CDS]
MKANEAFDNVCSLISGKYTASGWKYSKSGHWMTKKDKNFMYKVFFYTSWNNISDKNVAFYGECAIIPLKSKDKIFHINTR
RCNVPSGQLYWNIANEENWERTVKEFTNWLDSVFIPIVECCMNDLDNFVKQVAVRGFYPPNGYVVDISFILTYGSRELAE
EATKRYYASLDESIKKEFKENYESMINGNKAVSAYGNNMMRNYSNFRNIVENKIVVII
>lcl|NZ_CP026520.1_prot_WP_129112452.1_808 [locus_tag=PC41400_RS04225] [protein=hypothetical protein] [protein_id=WP_129112452.1] [location=943498..943719] [gbkey=CDS]
MKNISIIIALGIVLVSGCAKSPNYSNQGSNANNVQPLIQANSNNNQVDGSNNTKILMKRLLVTHFSLKTVSHS
>lcl|NZ_CP026520.1_prot_WP_206735005.1_809 [locus_tag=PC41400_RS04230] [protein=hypothetical protein] [protein_id=WP_206735005.1] [location=943716..944210] [gbkey=CDS]
MKYKGQFLFDDIVEKDVILKINVVDNLKYGKLYELKLDPIESVPNERLSLGYFYVQKDKIYKIEPTKDNLIKLKSSEELP
SGSVIVCQEQEIKDTLSKNEPGWHHYLEVNGDKREYHSFNNQVSTGYYESFIWESSKGLINYKSGYGAERDSMELQLDNN
NKHG
>lcl|NZ_CP026520.1_prot_WP_042232023.1_810 [locus_tag=PC41400_RS04235] [protein=hypothetical protein] [protein_id=WP_042232023.1] [location=944242..944697] [gbkey=CDS]
MSNKDIVRLIFDDRKDVEAILGDPEAFDIHECILNYGLSTKKFLYLDYKGEDDREIVDYMLDYEFARNIELASQEELEVL
GEFEYELLPDKIKEANKIISKRGYGLFSYPTSGDFYALFIAKLENKTKLLQVDLLHDERIPSKERYVQYYF
>lcl|NZ_CP026520.1_prot_WP_042232021.1_811 [locus_tag=PC41400_RS04240] [protein=DUF600 family protein] [protein_id=WP_042232021.1] [location=944747..945223] [gbkey=CDS]
MEQQMNDLYREIAETINQLIPEDWEDFYFNGEVENGEGGVFFFFRPKNGSEYIYSLDIPNRYDVKDEYDQLEAKLFEVTN
DLKNLFLENGQEPWFSITMKLTSEGKLNIEYDYTKWRESNFGPSDRLEYWESKYLNTVPQDETDRIKMDKMREFEGYV
>lcl|NZ_CP026520.1_prot_WP_042232019.1_812 [locus_tag=PC41400_RS04245] [protein=hypothetical protein] [protein_id=WP_042232019.1] [location=complement(945380..945841)] [gbkey=CDS]
MRVKKWVLGLMTAALLSLTLLISGAAAAPVTSVHSIWSTKPIQVIDNGTAYKAKAGNVIISWTNEPLSSGDSGFFNAIVQ
ANGSQYEFRLVSFGSSTNDEIVGKFDIYKNSTLVASVAGTAYGLSQPVGSYFKFYDTTQQWHVSAYITSRFDY
>lcl|NZ_CP026520.1_prot_WP_241688880.1_813 [locus_tag=PC41400_RS04250] [protein=hypothetical protein] [protein_id=WP_241688880.1] [location=complement(946129..947157)] [gbkey=CDS]
MKTALASSLTLLSLVVSLAPVSASSNNRELEKEVAAGITGQERQIMLDVMKNLPKAERENVIYIDDNGQIHANKTELKDV
FGKAAVSEENTVTVGNNETFTLPTSTDSPALNKNASVTPLAGYACGEAGGPYRRVFSNPGYSWWSGNVYLPSSTNNEVYV
NNASSNDTPYIYTGGHASNGASEVDAGFQYSKTYNNWAHSIFTGGVYYHNNSFRFKAGQNLTFKFYVPADGQVALYAAGI
RYDTNVADNYTVVTNASGWKKSGTGNELKRITAIASNASKPYNTGTYLKNVKWSNSMIGTSSTSNHQWLAADTGGYCTYP
NSTQVSVNYVNAGEETVSINMN
>lcl|NZ_CP026520.1_prot_WP_042232016.1_814 [locus_tag=PC41400_RS04255] [protein=helix-turn-helix domain-containing protein] [protein_id=WP_042232016.1] [location=complement(947394..948452)] [gbkey=CDS]
MYRMMIVDDEYMIHLSVRKMVETSGLDIIIAGEAEDGEEALRLLGSCSPDIVLTDIRMPELDGLSFIEAAKKRNERTQFI
ILSGYSDFEYARQAIRFGVSDFLLKPVDPDQFREALGRVYDQLASSETKLSRQHEWLLTLQGLSGELLEQIWAASEDKAQ
LLAGEALRHYMSSGPAELSPPQFAHHFTGMLEAELGKRDFRFARPYADREWPGHAADLGEALRDALSGMIREIKGTRNLG
SRQNIHRAMQYMQEHFAREDLSLSEVAEAIGMSDAYLSRTFKEETNVNFVKYLIRLRMEKARDLLEHTDCPTTDTAYQVG
FSDYSHFSKTFKKTYGLTPSELRKQSRQHPNG
>lcl|NZ_CP026520.1_prot_WP_042232015.1_815 [locus_tag=PC41400_RS04260] [protein=sensor histidine kinase] [protein_id=WP_042232015.1] [location=complement(948471..950603)] [gbkey=CDS]
MKIRNSQRKIVSIHRRLFILIIFFIALPVLVMGWLWYESSTRTIEQSAIETNRRFIEQTGKNLDLYIKNLENSTYPIVTS
PLTQQFLAPALPSAYPYYQLTEKVENDLFAQMIYGRSDIVGISLVGKNSLQITDFSEAKEILDMNTIRSRNADLLKRLDT
LGNFEILGLNRVGSSPVLTVARKLHSSSTYLYEGLLIVDLNLEQIENICKDESQGSFSTWISSADGELVYHPDAWKIGTR
ISGEQLMQIQGGEPASASASFRTGAPQGEELVIYETSRVTDWIVGANIPMKTIIGDLIRLRNISLAAALLLFMIALSIVG
GYSLSITRSLTYLQRLMAKVENGDFNVRLTSTRERKDEIGLVFRSFSKMVSELNRLVREVHSSKLKEQELIIKQKESALQ
SMQAHINPHFLYNSLEIINSHAILENNRSISKMTAALAHIFRYNTGNTGPLVSLGEETAHIRSYLEIQTSRFRNLQVEID
VEEPCGRTVPAVRLTLQPLVENAFIHGYRDKKPTYIGIIGKARETYYAVEISDHGRGMSADAAGRLTSLLAGVTGDVPSE
QGGTASVGPGEAPSAVPGRASAAVADLSKASLEAVAASAGRGAGASAGNGPGAAPAPVNDPVVPQRPAAGSAPAPEPLRG
PGTPQQPAAGRTWPDAAGTQPDDGARQGGIGLLNVHQRIRLTFGKEYGVFLLSSSPGAGTCFEIRLPYTN
>lcl|NZ_CP026520.1_prot_WP_042232012.1_816 [locus_tag=PC41400_RS04265] [protein=Ig-like domain-containing protein] [protein_id=WP_042232012.1] [location=950841..953114] [gbkey=CDS]
MTSVWNGWKKGFLFALVLLLGLVTFPVSRGNAASLLTIDSHRDGGRLEVGTVRIGGSYAGAYEVELVINGQKVADAHMDD
PDGDDAGTWYYDLDTTAYDGEVELIARGKDTATRYNAWSAYTRLTVHNPAAGVPQVTITSPADRDKVSGAAVVAVNVYGA
SPVKSVEVRINGGNWKRAEKKGNGYSYKWDTKKAGTRTNSIEARATDVRGHTGRSLTTYVQTGGGDGTKPVTVLPRQDRS
MWIWENASYNLIRTPGSRQVLDAMAKDTKTFGQDPVTTLYLGVDKLGGADMLEDERPEVRDFVRWAHAQGYRVQALIAGG
TVPPYFGTYPRYREDALREFEKVLNYNLASKSAEQFDGVNVDTEPYSLPDFKTGYPDVQIQYLDMLAALMERKEASGLSL
PVGPAIPRWYDSSDTAKSITWNGSTKWLSEHIQDTADYIAIMDYRDQADGSVGIIQQAQGEIDYANKIGKPNSVILGVET
KDIADGGDPETISFSEEGRSYMERELDKVYAAFEGNAAFGGIALHHYDTLRALPSAWGPGARFWQPPADNRPPSGVKGKP
QAAAFDYQRIDLTYGRATDNMEVEGYKIYRSTVRGFKPAEELLAGTARGLTFKDDGLLPDTVYYYRVAAVDASGNEGPAS
PEVSAKTGVTGLKPMIVDGMQVTYDGTRATVRLKTADMRTGEPVAAAVHGRFTRMGGKYVDGRSAADGTFTAVSELVQAD
SGEIGFAPRRVMAGGYYWAHAYDKPREASAVWGGAEE
>lcl|NZ_CP026520.1_prot_WP_042232009.1_817 [locus_tag=PC41400_RS04270] [protein=sugar ABC transporter substrate-binding protein] [protein_id=WP_042232009.1] [location=953382..954743] [gbkey=CDS]
MRRKIWTNVLLSALTLAVVGAGCSKGTAETTGTTAGGGSQEGGGKPVEILLGYYSDGKSDAKMKELIDKFMKKHPDIKVN
TQSAPYGQFYQKLDTQIAAGQAPDVWLSDGVYVMKYAERGAVKDLTEWIGKDLKKDDYYGLDFNKDAKGKYWAVPQGIQV
GALFYNKDLFDKAGVAYPNENWTWEDLKKNAALLTLDAKGKKADESGFDPKNVNQFGLTFFSITEGWFSVMKSYGGGVLD
ETAQKSIINTPQNKQAFEWMVDGMKRGIITDPADLKSFQSSMAVFPSGSAAMRIGIYARVLAANEAKLNYDVTLLPKGPD
GKRVSPVIANSWVVNQKSADAKAQAAWEWIKYWATEDDVQKEWAALGEAIPVKKSVANSDVFLKAGEKPANRKAFLESLD
FAKTLDNNAVWEEWVGKFNENAQRAFLGDAPIDAALADADKAVQKVLDDFYKK
>lcl|NZ_CP026520.1_prot_WP_042232006.1_818 [locus_tag=PC41400_RS04275] [protein=sugar ABC transporter permease] [protein_id=WP_042232006.1] [location=954777..955730] [gbkey=CDS]
MNDTVSGKAYAKAQTPARKLGKLDQEGKWGLLLVSPYLLHFIVFVAGPLIASLYFSFSQYDMMNPPKWTGLDNYGRLFGD
TLFWKSLWNTLYFAILFVPAQTILALVLAVVLNQKLKGLKLFRMAHFIPVISSWTVILYVSDAIFNSRFGFANSILASLG
LDPQRWLQDGKLVIPLLVLIAVWKGIGYIMVIYLAGLQSVPQDLYEAAEIDGAGTLRKFVSITFPLISSTTFLVVILSTI
STFQAFEQVYVLTGGGDISSAGGPNNSSLVLMLYLYREGFTFMRMGYASAIAWVLFMILFTLTLIQMKVQKKWVHYD
>lcl|NZ_CP026520.1_prot_WP_042232004.1_819 [locus_tag=PC41400_RS04280] [protein=carbohydrate ABC transporter permease] [protein_id=WP_042232004.1] [location=955788..956612] [gbkey=CDS]
MKHKRLPTILRYTAIILSSLVMIVPFATALMNSLKTYKEYTTVPPKWIPEIFQWSNYAEVLKLGNIGGYTLNSFTVALLS
VIGALLSCSMVGYAFARLRFPLKNALFVMVLGTMMIPPVVIIIPHFIIFNKLHLLDTLTPLWVIEWLAQPFGIFLMRQAF
MNIPKEYEESAKLEGCNPIQVYWRIFMPMARPTLATLAIFTFMGKWNEILTPVIYLTSNDKFTLPIGILSLGGQWFGNEQ
LMVAAALLSLVPILIVFLFAEKYFVQNHSASGIK
>lcl|NZ_CP026520.1_prot_WP_241688788.1_820 [locus_tag=PC41400_RS04285] [protein=sugar phosphate isomerase/epimerase] [protein_id=WP_241688788.1] [location=956630..957583] [gbkey=CDS]
MKIAAFSGALMEYSIQEAMKITANLGFDGIEIACREPHLSPQTSLPRVKEIRRLADEHVLDIPALGGYMGRFSESSDSEC
AAAYDEFLGLLERADYLETDMIRIFPGGPNAFLAADYHYEKAAHWLKKCGAEARRHEKKILLEIHNVSLIETVESSMRLL
EMIGEDNVGLIHDAGNMYISDTDFGRASVTGLGSRLLHVHVKDEKRIPEAGGPGTFRNVTRHGEEHFLQCRLGEGEVDHR
ELFEALAETGYGGWVTLECFAPYPPVEQLAHDRDAVRQLIRETASASAPAIASTTVSTTAPTSASPSSSSVSASGEN
>lcl|NZ_CP026520.1_prot_WP_042232001.1_821 [locus_tag=PC41400_RS04290] [protein=Gfo/Idh/MocA family oxidoreductase] [protein_id=WP_042232001.1] [location=957613..958644] [gbkey=CDS]
MTTPVTAQPVRFGLIGCGIIADIHAKGIRETPEAELTAVYDFVPETAVRFGQTYSAAVYESLEEMLARDDLDVVCICTPS
GLHAGQAVQAAKAGKHVFVEKPMAITMADLDDMIRVSGENSVLLSTVFPRRMSPAARYARELIRAGRLGKLSLCSGRVNI
YRSQEYYDSAGWRGTWAMDGGGAMMNQGIHTVDLLQWLAGPVASLYGKAKAVLRDIEVEDTAVSVLQFENGALGLLEITT
TAHLGPVHRIEIHGERGSLVIEDDDIVALKIDGEPVELPEFEPFKIIPDGHRIQIRDMALAVREGREPIVPGTEGRHSLE
IILGTYESARTGKEIVCQGTPVG
>lcl|NZ_CP026520.1_prot_WP_042232000.1_822 [locus_tag=PC41400_RS04295] [protein=hypothetical protein] [protein_id=WP_042232000.1] [location=complement(958760..959365)] [gbkey=CDS]
MKLNKVVKKQNIILTLLAGLLLISSLYYLFIYNSVKDTIINTSGVYASFESEDELFNSAELIVSGSTSQSFEERKKMITT
FSVTNAIQDFYTITDFQVEHVIKGNLTPKTTIQVGEPISYFQGLTGKSKITRENYTELKKSSNYLLFLKKNSGGLYFIMN
AEFGKYNTDHTDADDEKYAPKQTFREQMIQDKTYAPLKLDR
>lcl|NZ_CP026520.1_prot_WP_042231999.1_823 [locus_tag=PC41400_RS04300] [protein=formate/nitrite transporter family protein] [protein_id=WP_042231999.1] [location=959680..960471] [gbkey=CDS]
MEVQPLLQVEKLAMKKMKIYKQSKLRYLSRSALASMFIGFGVIVAFKTGSFFYLEHSPFTYPMAAITFGAAIILISYGGG
DLFTGNTFYYTFAALRRKLRWPEVIRLWLTSYGGNILGAVAFALLILATGLFREHNVNGFLLSVAEKKMNAPTLELFFRA
ILCNWLVCLAFFIPMSLKGDGAKLFAMMLFVFCFFISGYEHSIANMCTFAIALVVEHPDTISWGGVLHNLIPVTFGNLIG
GGILMAVMYHYVNKPFMDEHEEA
>lcl|NZ_CP026520.1_prot_WP_042231998.1_824 [locus_tag=PC41400_RS04305] [protein=hypothetical protein] [protein_id=WP_042231998.1] [location=960474..960779] [gbkey=CDS]
MKKRAQRCPLHAEAQRSPELSAVAWASSPAGADAALKARDANAPGLAQSCPQKGSPRLAGVSLFVRGYTGCAPGEPARAG
RADLASARKKTSLTGQTVKEV
>lcl|NZ_CP026520.1_prot_WP_042231997.1_825 [locus_tag=PC41400_RS04310] [protein=SDR family NAD(P)-dependent oxidoreductase] [protein_id=WP_042231997.1] [location=complement(960844..961587)] [gbkey=CDS]
MRFTDKVAIVTGGASGIGEATVRLFAREGAKVVIADFSDRGQTVADELKAEGREALFVKTDVTKEDDVKHMVAETVKQYG
KLDILFANAGIAKDGPADKLSWEAWQKTIDINLSGVFLSDKYALEQMLAQGTGGAIVNCGSIHSHVGKAGVTAYASAKGG
VKLLTQTLGIDYAQHGIRVNAVCPGYIDTPLIAGRTQAITDHLTALHPMGRLGKPEEVAKAVLFLASDDASFVTGTTLLV
DGGYTAQ
>lcl|NZ_CP026520.1_prot_WP_042231995.1_826 [locus_tag=PC41400_RS04315] [protein=LysR family transcriptional regulator] [protein_id=WP_042231995.1] [location=961766..962668] [gbkey=CDS]
MHLEQLEYIVEVAKTGSITSAARNLHVSLSAVSQSITHLESELGVVLFARSRMGAVPTAEGKVIIGKAFEVLSKLDELKE
AARVWGETLSGELRIGTIPGPMSLLVDAVAGFKQDYPLVRMEIAEKGSQEILDDIRHNRLDIGFIILFESLLKKSEGVTF
ERLLEGKMVVGMSDKSPLAYRGAVTPEMLRTQPVVLYNDEYVRWFMNRFAAEHGEVDILFTTNNTDAIRKAVEEQNAVTV
GMDYSFRREWAPLFTEVNMLTAELEFKAGQPVYLGWVRAEGRHFPRVSKTFVERLRNGLR
>lcl|NZ_CP026520.1_prot_WP_009675738.1_827 [locus_tag=PC41400_RS29645] [protein=hypothetical protein] [protein_id=WP_009675738.1] [location=962890..963060] [gbkey=CDS]
MTYAEMLGRRSEILKRTIGDMIAKDNTKGLGMQESSFLRNMIKELHQNEYELQRNS
>lcl|NZ_CP026520.1_prot_WP_042231993.1_828 [locus_tag=PC41400_RS04320] [protein=hypothetical protein] [protein_id=WP_042231993.1] [location=963161..963640] [gbkey=CDS]
MNVKIIGVALYIMLIYGISKHYPAMHGLFFPALGAFSFLFISRPFNAKAMSSISAGAVLATVVGTLIYYLVPGILALFVN
TLIIIWLNKVKWNAPPILAVSLIPFFTETPRIWDVPLAVALSLAGLTVTLLAADFVHKLLMVTFRSGSKLAPGSEEAAG
>lcl|NZ_CP026520.1_prot_WP_042231989.1_829 [locus_tag=PC41400_RS04325] [protein=ABC transporter substrate-binding protein] [protein_id=WP_042231989.1] [location=964414..966138] [gbkey=CDS]
MKKRASYAVSLILAASLALSACSSSGTPGASEAPKTDDQAKQQTGANQFIAATDKSKSPAAATARKDTVVVTLTEPGGVF
NPYFYHNGYDGNVTSVMFSPLVDIDETGKPIPALAEKWDISQDGLKYTFHLKKDLKFSDGSPLTADDVAFTLSVLYDKAY
DGETDITETAVKGGQAYKEGKASTIEGIKVIDPQTIEITTEKANARALLLLGGQVLSKAYYGKNYTPGNLEYIKTLHTKP
VGTGPYKLEKFIPGQEVRFIANENYFGGKPSVEHFIYKTTSGDATQFFQTGEVDYSAYPANDDSLELLKGLGYANVNIYT
STAYSFIKFNHSKPIFKDKNVRQAFIYGLDRDKIVAAAYQGYGQVANVPISPASWAYTEEGVNTFKFDPEKAKKLLDDAG
WKPGADGIREKDGAKLVVHYFTTKGPLSDVLVPIAKENYKNIGIQLETELMDYNALLARTSKGDHDLASFSTTMLPDPSD
GVISFYSKNKSSFTDTNGYANPKVDELIEKGISSFDIEQRKTTYKELYKELSDDPPFILLTYRKIISAHNARIEGFKPNG
YTGISSSLPKLKIE
>lcl|NZ_CP026520.1_prot_WP_042231988.1_830 [locus_tag=PC41400_RS04330] [protein=ABC transporter permease] [protein_id=WP_042231988.1] [location=966214..967170] [gbkey=CDS]
MKSYVIKRLIYMLITLIGASMLIFMLYALTPGDFVDSNPKLSMERKIELKELYGLDKPVVERYFIWLGNLLKGDLGFSLQ
YQQSVTSLVVSFIGNSFLIAFTSMIFTWLIAIAIGVFSATKQYSWFDALITFLVFAAMSLPAFFVGLLIIKLLAVDLGWF
PPGGKITTGSTTTGLAYVWEVIRHMALPVAVLTMLSVGSLTRYFRTNLLEVIKQDFIRTARAKGLKEKTVVYKHALRNAL
LPAITLLGFEIPSLFGGAIITEKIFNWPGIGQVYMQAFSVRDYPVLMGFTIFIATLTVISNLIADLLYRVADPRVRLK
>lcl|NZ_CP026520.1_prot_WP_042231987.1_831 [locus_tag=PC41400_RS04335] [protein=ABC transporter permease] [protein_id=WP_042231987.1] [location=967183..968112] [gbkey=CDS]
MSVTAQHPLNPAAGQPSAPKSSLWKQSFLRLRKNKMAMLGLGVLIFIFLLCFIGPFFSPYSSGKVDAGLIKAPPSLKHWL
GTDQLGRDILTRLMQAGRISLTVGLASMVLSVLIGSVLGILAGFYRGVVDQVIMRVADVLMTIPELPMLFIMGAVLSEWK
VPSDYRLYIVMIMLSLVGWPGLARMVRSQILGLREREFIQAADALGLRDRRKLFYHLLPNTIPLLIVVATLRVGGAIIGE
STLSYFGLGVVPPTASWGNMIDAANSLIDFQKRPWLWAPPGFAIFITVIAVNLLGDGLRDAMDPKMKSR
>lcl|NZ_CP026520.1_prot_WP_042231986.1_832 [locus_tag=PC41400_RS04340] [protein=ABC transporter ATP-binding protein] [protein_id=WP_042231986.1] [location=968118..969098] [gbkey=CDS]
MNKPLLEIKQMSTHFYTDEGTVKAVDNVSLSLKEGETLCIVGESGCGKSVTAMSLMRLVQEPGGKTVAGEVWFDDKDLLK
AGPKDIRHLRGNEIAMIFQEPMTSLNPVLTVGEQMIEPIIEHQLVSRKQAKKQALELIAMVGIARAESIYDSYPHELSGG
MLQRIMIAIALSCNPRLLVADEPTTALDVTIQAQILDLLRKIKEERRTSILLITHDLGVVAEMADHVVVMYAGQVIEESP
VLDLFRNPKHPYTQGLLRAKPVVNQRKERLYTIPGQVPKLVDLQESCYFSDRCEHCMEICRTKQPPMLETGTGHRTACWL
YEEGIR
>lcl|NZ_CP026520.1_prot_WP_042231983.1_833 [locus_tag=PC41400_RS04345] [protein=ATP-binding cassette domain-containing protein] [protein_id=WP_042231983.1] [location=969095..970060] [gbkey=CDS]
MSTKLLEVKGLKKYFPVSGGFLNRTVNHVKAVDDISFTINKGDAFGLVGESGSGKSTTGRTLLRLTDKTAGEVIYKDRDI
FSLSKGEMRTLRTDMQIIFQDPYSSLNPRIRVGDAIGEAVLDHGLASKAEIRERVLEIMSVCGLAPHHIDRFPHEFSGGQ
RQRIGIARALILSPDFIVADEPVSALDVSIQAQIINLFSDLRDQKNLTYLFISHDLSVVEHLCNRIGVMYLGSLMELAPR
DELFANPLHPYTRALLSAIPIPDPEIRRERIVLTGDIPSPVDPPSGCKFHTRCPFAKQVCKEAVPEYRNVGGEHYVACHL
V
>lcl|NZ_CP026520.1_prot_WP_042231982.1_834 [locus_tag=PC41400_RS04350] [protein=winged helix-turn-helix domain-containing protein] [protein_id=WP_042231982.1] [location=970228..971139] [gbkey=CDS]
MAYEVEFSFQPANELIVSLHTYLCKIWYKRIDLGPDWAPEIRSGLTDDFRQELDSTDLSPEWRLIFLFAYLSPEKKDQPG
FLHWLESLSLGDLYVLLSPYIQNFPADMLAVRDKFVHLLSEWNSQYFAKVEPGIVTALEEDAEKNRALIGRMPAVELAEQ
TTSGMYFEPGGSLAKVVLTPQFHFVPGNVFYSFGPVTLCQYPASLPGHDHEEGPSPKLYRMLRSLSEKSRLRILRFLRSE
PHSFIEVVRYLGISKGITHEHLFSLRCAGLLKAHVTGETVTGYSLRTSAVGELQELLLEYLED
>lcl|NZ_CP026520.1_prot_WP_042231981.1_835 [locus_tag=PC41400_RS04355] [protein=ABC transporter substrate-binding protein] [protein_id=WP_042231981.1] [location=complement(971440..972426)] [gbkey=CDS]
MRFTGKNVRLGGALLLLLFSAACGTAGQQSGSGQPAAEGADGASGQAAVRTIRDAGGEVAIPVKPLRIADVSGSTEELLV
LGFTPVVTGNTEMSAPTQLTPILQKKLGGGTVKAGWFQSEVSLEAIAAANPDLILAGPAQSKIYDQLRTIAPTVRVPYGF
NAFRERFSFLAETLDKKKEMEDWSSRYEADSRKLKEQITKLTGNETFAVIEATQKEIRVYAETGIADMLYRDLGLPKPPG
TPDPDPWGGKVLSLEALPAIDADHIVLLADNEQNALEQSRLWGGLRAVKTGHVYRVTSRQSYNEAFFALGKENLIGQIAG
DILEKNKR
>lcl|NZ_CP026520.1_prot_WP_042231980.1_836 [locus_tag=PC41400_RS04360] [protein=AraC family transcriptional regulator] [protein_id=WP_042231980.1] [location=complement(972546..973514)] [gbkey=CDS]
MNQGIGQESFSIFDTYFEAMTRHHGTNGTDRTTGTDGKSGHMSSRLEIVRTDWTIRREREVRCISESAMVELSFCFEGCG
QVEVSGGAHDLQEGSCSLQLMRDFSASFRYEQGTRVRTLALGMPVSLFNAYHSSVSAAAVNPFNRLLGTGSFRQFRCPIL
PATRGLLQELEDCPYTGAMKDMYLEAKALELFAVYTNRFLYENDSQAPVSGTLSRTDRQKIREAREEIIRRMDAPPSLLE
LSRIVGINDYKLKLGFKELYGTSVFAYLREQRLHKACELLRTGDMSVSEAAARVGYMNFSHFAEAFRRKFGVNPSVYRKE
PR
>lcl|NZ_CP026520.1_prot_WP_042231979.1_837 [locus_tag=PC41400_RS04365] [protein=HAMP domain-containing sensor histidine kinase] [protein_id=WP_042231979.1] [location=complement(973741..974814)] [gbkey=CDS]
MRKSSAARVRTAVLILLTATAIMLLCTYAAYTGLPYLYSLLGAEPAPYWRQAIFATVFFILFVGCVGAISSLMSKRHMIV
WDTIQDAMRRIAKGDFSVSIDLPMDEKDRLGTIVSNVNQMAVELGALEKMRQEFISNVSHEIQSPLTSIQGFAKAIKQEN
LSPEARLRYITIIETESERLSKISDNLLKLTSLESKHHPFEQIPYRLDKQLRNVVLASEPKWLEKKLDMEVELEPLAVRA
DRDLMSQVWINLLSNAIKFTPEGGTIRIGLSAGEGRARVSVADTGIGLTEEDRLHVLERFYKADKSRTSSGGGSGLGLSI
VQKIVEMHGGEISVESRLGEGTVFLVTLPLEVQEPDA
>lcl|NZ_CP026520.1_prot_WP_042231977.1_838 [locus_tag=PC41400_RS04370] [protein=response regulator transcription factor] [protein_id=WP_042231977.1] [location=complement(974811..975488)] [gbkey=CDS]
MANILVVDDDAHIRELVLIFLQAEGLQVIEAVDGLDALSKLETHKIDMVILDIMMPKMDGWQLCREIRAHHGDLPLLMLT
AKGETSQKVKGFNLGTDDYLVKPFEPAELTVRVKSLLKRYRISVSQAVEVGSVTMRRSTYELLWNGENLTLPLKEYELLF
KLGSYPGKTFSRDQLIEEIWGYDYAGDERTVDVHIRRLRERFPEEDCGFRIRTIRGLGYRLETAE
>lcl|NZ_CP026520.1_prot_WP_042231974.1_839 [locus_tag=PC41400_RS04375] [protein=ABC transporter ATP-binding protein] [protein_id=WP_042231974.1] [location=complement(975539..977293)] [gbkey=CDS]
MQETIQDKKQLGNWRLFIRMIQDTHPSKLLLIAALVMSVATTGVGFVVPLFTKSLVDGFSIQSLNPWQIGLLVLAFVLQA
VASGLSIYLLNRVGHHVVANLRDRLWKKLLHLPISFYDTNESGEMLSRVTNDTAVVKEVITEHLANCITGVLSIIGSVII
LLFLDWKMTLVMLLAVPLAMVVLMLLGKKMFTISRGMQEETAKFTAVLNQVLPETRLVKASNAEEIEYERGKTGITNLFR
FGLREAKIHALISPLISFVLMVVLVAIIGYGGVRVSSGALTAGDMVAFILYLIQIIIPVTQLTMFFTHLQKAMGATERMM
SVLEQDEEKLYEGTALKDVSRPIHVKDVDFAYGDERILTGLNFTIEPGKVTAIVGPSGGGKTTTFSLLERYYKPTGGAIT
LGDTPIEQFSLHSWRSHIGYVAQESALISGTIRDNICYGVEREVTDAELNKVVEMAYASEFIAELPQKFDTEVGERGIKL
SGGQRQRIAIARALLRDPQILMLDEATSSLDSQSEIYVQKALDNLMKGRTTLVIAHRLSTVVDADNILFIEKGHITGSGT
HDQLFQTHSMYKEFASQQLRLQEA
>lcl|NZ_CP026520.1_prot_WP_042231972.1_840 [locus_tag=PC41400_RS04380] [protein=alkaline phosphatase family protein] [protein_id=WP_042231972.1] [location=977573..978592] [gbkey=CDS]
MLHARNDWKKTVSAALLAGLLLPASLAGAAASPETSAAPSIAGNAAGGPESSAVQPDALKPEAGQAAPAAAVKKKKVLVI
GLDGTRPDALQAANAVNLEALAANGAYSWTTKANSNYTWSATGWSTIHTGVWYQKHGVKDNTWTGSKFGQYPSLMKRAEL
YNPALSTASIVHWGPINDKLVEGIDLEKTVATDAEVVSDTVNLLKTGNPDFLFLHFDDIDHAGHTYGFSPKVSQYTKAIE
GVDAQIGTILDAVKARSTYAQEDWLILVSTDHGGKGTSHGGGSSEETTIFLIASGDSTVKGQLTGATYQADVMVTALKHL
GVPVSSSWNLDGRAVGLIQ
>lcl|NZ_CP026520.1_prot_WP_042231971.1_841 [locus_tag=PC41400_RS04390] [protein=extracellular solute-binding protein] [protein_id=WP_042231971.1] [location=979089..980681] [gbkey=CDS]
MTRKLLSTFLSLTLFATIASACAGGKQEGASSANTKPSTAAAADYGDTGGLKLPLVDEPVTLNWMLVSDNPDLSNKLIVK
EIEKRTGIKLNIQAYSTANYKDKLKVVMASGKLPDVFHGLTPAEANKLGSQGALTPINPYLDKLPNFKSLYVDNKDSSWV
MKSWSDDKDNLYTWPIYNINRDVNHGFLYRKDIFDKLGIQEWNTTEEFYQTLKKLKEAYPTSYPYASKTKDYIFRDWAPG
WGIGGASYPAAYDENTKMWKLAFIQPEYKQMLDFMKKLYNEGLLDPEFITDTEASWTAKMTTNNKSFVTFDWIGRLDLFH
NQVKSQNPDYNLRYGNPVGPTGKIKQLSKIDNFGIVVSNNKNKEVALKLLDYLSSPSGAALVTMGVEGETFKMEGGKAVY
PELKDVPNVDIGVLKDRYGLWLEGMYVRADSRSVYYNFTEKEQEAQDKMVKNNKIDVLDPVLKFTDPETKKIADVLTSLQ
KSANEFSTKYILSKSYGDAQWNEWLQTAEKMGASAYIEAYNNAQKRFDSK
>lcl|NZ_CP026520.1_prot_WP_042231970.1_842 [locus_tag=PC41400_RS04395] [protein=ABC transporter permease subunit] [protein_id=WP_042231970.1] [location=980769..981734] [gbkey=CDS]
MHQAQSELAQTAVSQPKRSKGAEILRHIKRDRQLLLLFLPCIVFFILFRYGPLYGLIIAFKNYNVYEGILGSEWVGLEHF
KTFFSSQDFWMLFRNTILLGLFSLLWGFPFPILFALLLNEVRLSFLKKTVQTLSYLPAFLSVVIISSMIIDFLSPTRGLI
NTIIEALGFTRTYFLIDPEWFRTIYVSSEIWATIGYDAIIYLAAIAGVSPTLYEAARVDGCTRFQMMRYITIPSILPTIT
IMFILKTGAMFRIGYEKVLLLYNPMTYDVADVFSTYVYRKGLLETNYSYAAAVGLFEALVALVMLLVSNMISRRAGGSSL
W
>lcl|NZ_CP026520.1_prot_WP_042231969.1_843 [locus_tag=PC41400_RS04400] [protein=carbohydrate ABC transporter permease] [protein_id=WP_042231969.1] [location=981728..982606] [gbkey=CDS]
MVTAKGSKLRKFGLFDLMNVIVLSLVAAATVYPVLYIIAISFSDSAYIVQNKVFLWPRGFNLDAYKEILQNPRIPRAYLN
TILYTGAGTLINLLMTAVAAYPLSRKTFFGRKYFMIAIVITMFLNGGIIPTYIVVQKLHLLDSIWALVLPNAIWTIELLI
LKSFYESMSSSLRESALIDGASEYRILFSIIIPLSKPALASIGLFYFMGHWNSFFLPLIYLNDANLYPLQVVLRDMLIFD
QAKESGLVDQAALTPEAMKNATIFISMIPVLLIYPFAQKYFAKGIMLGSEKG
>lcl|NZ_CP026520.1_prot_WP_042231966.1_844 [locus_tag=PC41400_RS04405] [protein=helix-turn-helix transcriptional regulator] [protein_id=WP_042231966.1] [location=982685..984964] [gbkey=CDS]
MPRIRWKANMGLLKIFFSLLLVAVVMFASNYLVFENSISGIYRQVSENNKLVVKHVIRSFDQAFKEMNDLIYSIHALPYK
SWEMMEDDRINMNDVYLMQKNMTTLVSSMDYLEEVVLFEKGSAIAITSQGTSSLEALFNEQYRHDTYNAEYWRSFAETNR
TQKVFPASVFKVQFADRLVQNRNLIVLMGSNQISTYNIMLLIDVDKLLKHVNQTAMMQGTSLMVMDAERNRILSTDPEWD
LVEMLNEVYFNPGQEATVKSRDYEYLFYKSDYNDFIYIDKMPVELAGTAPVVAWNRMIMIIAILSAVGLSIVLSHYLYRP
IRDLVRLLGGKTAEPQADYSRIRSGIEQIQLENASVKVQMETVHSEIRRGVFLHALDEFSHSREFENRMQHIFADFFRTG
QFMMAAIHLRPKEGPEAGSEPPAPLRAASLEQALQAVLREAAGEAGGHCAVFHAGSLQFLALIGLDRPSDREPVIRRMGQ
ILKQAGNGPLKDVAVMAAVSRLYASKIQYCQDAYREVKRGLMYRDLSEATPVTDTQAIRYTWKVYFPLHEIEKLSNHLAG
GNVEACLAMIRAILEKNAERHVHHHQLVPVAECIYYHMQQYVEPGKVDGQELIRLQEGFIRSVHQAVNYREIETALDYAA
RRFAEWRKPAGDTKLEAEFIAQYINTHYKENLHLDHMAELLGTTPKYFSNYFKKAFGINFVEYLNKVRMAHAKEWLKTTD
WSVAEIGEKSGYLNSSTFTSTFKKYNGISPSEYRRKSIS
>lcl|NZ_CP026520.1_prot_WP_042231963.1_845 [locus_tag=PC41400_RS04410] [protein=zinc-binding alcohol dehydrogenase] [protein_id=WP_042231963.1] [location=985395..986393] [gbkey=CDS]
MKCVAAQHGKIVLRELEEPPLLPGHILIRTEYSAISPGTELSMLRRSGETPVELGYSAVGIVTQAGEGVRGLEPGQRVAC
YGAPYVRHAETLAVPANLAAPVPDHVRPEEAALAGLGAIAIHALRTADLRFGESVVVVGLGILGNLTAQIASAAAIRTAG
LDLDAGRAELLRMGGVEAFASADELEAQLARVTGGLGADSVLLCVGGDGEPLINRAFGWLRDRGSVVLVGDVSANFSRGQ
MFGKEARVLISRAGGPGRYDPMYERDNRDYPVGYVRWTEGRNTAEYIRLLAEGRISAAPLLTSLPFSRAAEAYESFRGSE
PALGVLLRYGAD
>lcl|NZ_CP026520.1_prot_WP_042231960.1_846 [locus_tag=PC41400_RS04415] [protein=DedA family protein] [protein_id=WP_042231960.1] [location=986814..987404] [gbkey=CDS]
MGDLILELVDQYGYAFFYAALALGPFGIPVPNEVTVMTGGVIGAEGLLNPWLVYLCIFLGLITAVTIGFVIGRVFGKRLI
NRLRGKERFARYFVSSEQLLDKYGTAAICISCFIPCVRYIVPIFMGAYGKRYAKFALYAYSGAWVWTLLFYGIGRQWGGN
LSQLLIVPETKLFLVIGCAAAVLFLFWRNRTKKSQT
>lcl|NZ_CP026520.1_prot_WP_042231958.1_847 [locus_tag=PC41400_RS04420] [protein=MerR family transcriptional regulator] [protein_id=WP_042231958.1] [location=987524..988285] [gbkey=CDS]
MLSDKLLTVKETARVSGVTVKTLHHYHKIGLLPPREISEAGYRLYGMEELQRLQEILFYRELDVPLEKIGRVLEEKPQRT
AVLSEQEELLTARKVRLEQLLLTLRQSIRYASEGERMEMADMFKGFGKEEEWKEALTEQNEYLKETYGYDLLEKETIDVE
ALNEQAAEAKRFTDGMAAALKDGVKFDHEDVRTLLRDHLDFLNKAGHAFGPQDYVNQAKFFMEDDFHRQMLESQQTGLAY
YLRAAAESWASAQ
>lcl|NZ_CP026520.1_prot_WP_042231956.1_848 [locus_tag=PC41400_RS04425] [protein=hypothetical protein] [protein_id=WP_042231956.1] [location=988433..988738] [gbkey=CDS]
MAFGVKREELQRWKAAVSRGDIAYLTHYWLEPRFPGITTVTKVGCSDLDRLSDWCISHGLNPRYIHNRAPFPHFDLIGTK
QKEILRKEGIHDQLERFRLLD
>lcl|NZ_CP026520.1_prot_WP_042231952.1_849 [locus_tag=PC41400_RS04430] [protein=beta-galactosidase] [protein_id=WP_042231952.1] [location=989257..991062] [gbkey=CDS]
MTSFKVKGSQFLLNDKPFRIISGALHYFRVVPEYWRDRLLKMKACGCNTVETYVAWNVHEPEEGKFEFGGIADVTAFVEL
AGELGLHVIVRPSPYICAEWEFGGLPAWLLKDSEMQLRCSDPKFLAKVDAYYDVLLPKFVPLLCTNGGPIIAMQVENEYG
SYGNDKAYLRYLRDGMIDRGIDVLLFTSDGPTDEMLQGGTLPDVLATVNFGSRSEESFAKFREYRPDEPLMCMEFWNGWF
DHWMEEHHTRDGEDAARVLDDMLGAGASVNFYMFHGGTNFGFYSGANHIKTYEPTVTSYDYDAPLTERGDLTTKYEAFRE
VISKHEGEPECALPDPLPVKEYGEVEMTERADLFAQLGKLSQPVRRVTPEPMEKLGQNYGFILYSTHVTGPRRGQELHIQ
DVRDRALVFLDGSYIGAVERWDVRPLKLDVPAGGARLDILVENMGRVNYGPLLRDHKGITEGVRLDNQFQYGWDIYPLPL
DSMDGLEFGTATAAQHADTAGEKPAFYRGFFEAEEAADTFLRLEGWTKGVAYVNGFNLGRYWERGPQKSLYVPGPLLRKG
TNEIVLFELHGTEQLSVRLEDKPDLGTEAAVDQSILNFTGE
>lcl|NZ_CP026520.1_prot_WP_241688789.1_850 [locus_tag=PC41400_RS04435] [protein=hypothetical protein] [protein_id=WP_241688789.1] [location=991618..994713] [gbkey=CDS]
MPENQPNRQIFSDRSGSVIPWNQVKGSRKVEYSATRRRGVDLNVGAAEKFVAKMKNVPVFAQIGGEVTFRAQAQAPLDLF
DEAGLAVRLQAIARLAAGIGLRPRMSVRDLTDAVGGLPDMKGAPMELFQLLLGEITIQGEMHAQAAVAIMAYANLVASGT
FLSNDGFGKPGFQYIFEAGYGFKVGGGYRAFVEVNFDNPRRLINRYAEAAITQMIALAGQSTSPEQLALIRSLEAPVIIG
ARLAYEIGELPREQAGERSIAVIILEGQRWLLKRFRETASNSTVQWTANNLLTSLNKHMESKMSLYPFNGPLEVQVPSNV
ANDVNTQLGRSPGTPLSQTDLITYLTGKPLQTLLSQSEDTRLILNIFQNAFPGNPQTSAGAIFAYTGSNPSGLLTGLTSG
LQAYAGQQLNTPAFKQAVQQVASGSREMGVILQSGLIPSLEIAASVALPYVMNQGGYDRTFVREALSSALLAIVGRGLVR
MFNGVTMHIQGQLGGLLNGAANQLSSPAVVNLLELTGPANRYLIRPLQAGFREAADQLANFIPQRAFDQMLIIFTPIEAG
STASFVQNLRKTDWSPRAGNLSDVANTLLTILMTRATTFTSKIIAVVLQAFLEVLLQELQDLIDKFGRILKEAWEALRNY
MRTVLTEPLRRIIRELVGQKISVLPPRIQSLVTEGIDAAINSLLPGAVLDPIMNGLSKVTLDSKKVLDTMTGQGNFPAFT
DYLFQEILQQLRRSNLNLNLSINVPIKLPGPLSGPAFTGGGPAGYGGIGGLGSGSIGDLGGGLGNIGGLGDAIGGLGDAI
GNLGGILKGNESLGEALDKILNNAGLGGLIPVPGGNPQIPLGVTLTPEHIVRAIREAVPIRTLQGQYRNLRSQAAPALQK
MQQVAEFQAAIPRIVADFQINRSSERSLGIVYTDALILLRPLPGQGNPSVFELLVHFPGLRKPVPGIIGTYPQISVVLDN
CQLPLNTFQTRWNDNGGIQGRPPGLTLRRTLTLRELRGGTNMIYTAASAPYFGKVEAATQFRIQSNPPLTK
>lcl|NZ_CP026520.1_prot_WP_042231947.1_851 [locus_tag=PC41400_RS04440] [protein=alpha/beta fold hydrolase] [protein_id=WP_042231947.1] [location=complement(994829..996718)] [gbkey=CDS]
MAHIFITGGTGFIGRHVLRALSLPGAGSGGSHRLTVLVRSLARFDSILQKEGLEYSPHIRPVKGDLSLPSLGLDDIGLEL
AGQADILLHMGGPMDIGLGREEAESVFFRSSEEMVSLAKQIHTNRGLKHFIHLVGFKSPYHEKNADLPEHIGHPGEPPYE
NRKFAADLYIRRELYREGIPLSVVNPSVVIGDSATGVTEQTGGLGILVDSVRRGLMPLSPGGAGHWLPLVHADHAAAFIA
ALAREEEPGNGTYFLLDEKSGSPDMRELARSIAKEVRVRRPFGAMPLPLLSKALGTKWGRKTGIPAESMNFIADGDFPLA
SKQAVERKYKLDTGLSRDILPGTIADLDYRLSHGEAGLPGAEFKPGRLGKLAVWSRQRGSGIPVVLLHGTFSGAFPLLPA
ASRLNGHPVYAVDLPGFGRSPYHHGSDVIAGYVEAVTGALMELGEPVVLAGHSFGGLIAAKVMERLPGYVRGAVLLQPAL
HPAHAGVPAWIAGAYMKRMTERALVRKLLESGSFADEEEIPQEYVRYVLRDLASPRVRKTNADVFAALGNAGAFGLSPGK
WDTGKVRLIWGELDRRYRLPQAFRHLPVTEIPAGHQFPISHPDRTAGLILAHAENCGSERTAMERQPLF
>lcl|NZ_CP026520.1_prot_WP_042231944.1_852 [locus_tag=PC41400_RS04445] [protein=TetR/AcrR family transcriptional regulator] [protein_id=WP_042231944.1] [location=complement(996781..997362)] [gbkey=CDS]
MKTTKRTSNRDKALNTAEHLFLTKGFLLTSMDEIVTESKVSKTNIYYHFKSKEELLGAITERLVGRYEELIREAGSRSGL
TVPQKIAAIGDLLMRGEDDFLGGCPFLTLYVQTSREAGFVRERIGRFFREQTLYLEGLLAEGVRSGELRSSLEPGPAAAL
ILSGIEGALFLQHASPDSTYMQQVLPALAQLLK
>lcl|NZ_CP026520.1_prot_WP_042231941.1_853 [locus_tag=PC41400_RS04450] [protein=polysaccharide deacetylase family protein] [protein_id=WP_042231941.1] [location=997427..998332] [gbkey=CDS]
MSLAEALGYRPDEKLLIVNADDYGMCHAANEGIAQLLAEGAVSSATVMMPCGWAKEAVEWAAAHPQTDVGVHLTFTSEWT
PYKWGPVTRHGSVASLVTQEGYFPADSASFERRADPEEVRTEIVSQIELAMRMGVMPTHLDNHMGSLYGLGTGRDFLGIV
FDICASYGLPFRIPRYVDMGLEVPAEAAKTAGLRAAQADAKGVVILDYLLGLPFHMEKGETYESFRGSVMRLLRALKPGV
SEFILHPALPAEELKAINPHWQKREWEFRLFRDPEIKGLLAAEGIRLIGWSELQRLQRASR
>lcl|NZ_CP026520.1_prot_WP_042231937.1_854 [locus_tag=PC41400_RS04455] [protein=DUF4183 domain-containing protein] [protein_id=WP_042231937.1] [location=complement(998403..998831)] [gbkey=CDS]
MPPSLIKLFVAASSVITGTVTTTTTTAVNDNVRRFDAVVTAPMIDAGAGTTTIPDTSFRDDSGTTLAAGALPVPPAGGYY
NLYANGVLQEGGLSTLSATNLVIATAALLPGTPVIVEVHDYTGTTSASASTPNLTVTTTVNT
>lcl|NZ_CP026520.1_prot_WP_042231936.1_855 [locus_tag=PC41400_RS04460] [protein=DUF4183 domain-containing protein] [protein_id=WP_042231936.1] [location=complement(998854..999240)] [gbkey=CDS]
MRLKRKKRIRKTKQSGSLRILQAKTAVFVTFSDGFRRIYRNRDQDPSYGRVRIPDPRTVSYTNLFVNAVLQPSSVYRVSR
GKLKLRSPELPEKGVPIMLQFVTILGKHSRGCRGRKTATTCRSPHKVR
>lcl|NZ_CP026520.1_prot_WP_042231933.1_856 [locus_tag=PC41400_RS04465] [protein=hypothetical protein] [protein_id=WP_042231933.1] [location=999418..999894] [gbkey=CDS]
MKAGWAALLFLLVLTACQDGREVQVPREHSLAKSYLEDKGYRIMSYEGEGQSYELTKDVLRKLPGMMYWGLQTVDPEQYL
GKLIRVQKFVVKGHPLSKGKVDVYVYESEGEPFGGFSFPQDQYGSTGGGWSLDGKTLEEVHSVTYPDWRAAWREKYGN
>lcl|NZ_CP026520.1_prot_WP_042231931.1_857 [locus_tag=PC41400_RS04470] [protein=DNA-3-methyladenine glycosylase] [protein_id=WP_042231931.1] [location=999908..1000459] [gbkey=CDS]
MDASFLERDTVTVAKELLGCELVRQTEAGLIRVAITETEAYQGSDDPASHACRAVTPRNRLMFGEVGRLYVYLIYGMHLC
INVVAHEPGGVGAVLLRAARPLEGLELIRDNRGGAPDRNLLNGPGKLAQGLGITRDLNGCDLLAEPPQGLALERGAPLAG
PILVTRRIGITKAVDYPWRFVAE
>lcl|NZ_CP026520.1_prot_WP_042231928.1_858 [locus_tag=PC41400_RS04475] [protein=alpha/beta hydrolase] [protein_id=WP_042231928.1] [location=1000865..1001806] [gbkey=CDS]
MPVEPRIAMMLKQMNSHPLPSWEDLRPEDLRKGTGNAAFTFGVEEVKQVEDRKLPLKGRDIPIRIYTPEGRCPWPAFVFF
HGGGFVVGDLESHDSICRNLANSVHARVISVDYRLAPEHKFPAAVDDAYDALQWIASHPDEFGIDPARIAVGGDSAGGAL
AAVSCIKSKEAGGPEIVYQLLCYPAAGFLEEDPASLRENKEGYLLTVGMMEWFSKQYLNTEEEIRNPYAYPIHYKDFGGL
PPAMIVTAQYDPLRDSGKAYADKLIGAGVEVTYKNYETLIHGFANFHKFVPAAQEALDEMASQLRLALGTDKR
>lcl|NZ_CP026520.1_prot_WP_042234209.1_859 [locus_tag=PC41400_RS04485] [protein=cytochrome P450] [protein_id=WP_042234209.1] [location=1002096..1003289] [gbkey=CDS]
MNALFSPDFLRNPYPLYAQFRQNQPVLFMEPLGIWTVFDYEHVKQVLFDHTRFSSEFRPKFEEGTEQEQRGGGFSLITMD
PPRHTQLRTLVSKAFTPKAVAALESRIAAITHELLDQVSETGRIDLIRDFSYPLPVIVIAEMLGIPSEDREQFKHWSDEV
VASADSVVGATNSESQRAHEEMNAYFRKIIALRREEPRDDLISALLAAEESNTHLSEGDILSFCALLLVAGNETTTNLIG
NAVLSLLENPDQLAKLRAQPELLPSAIEEVLRFKSPLQAMFRTAARDVEIGGQVIPAGSRVVAWMGSANRDAGKFADADK
FDIARESNPHIAFGHGMHFCLGAPLARLEARVALQAVLERLPDLARADQDPLTPARGFIVHGVSSLPLTFTPTNRLA
>lcl|NZ_CP026520.1_prot_WP_009674668.1_860 [locus_tag=PC41400_RS04490] [protein=cold-shock protein] [protein_id=WP_009674668.1] [location=complement(1003468..1003674)] [gbkey=CDS]
MHYSKKNMEPVPEEEITVWACSQESCVCWMRENFSFNQVPLCPICKSEMVKQVKMLPVLTNHNHNALG
>lcl|NZ_CP026520.1_prot_WP_042234207.1_861 [locus_tag=PC41400_RS04495] [protein=cold-shock protein] [protein_id=WP_042234207.1] [location=complement(1003803..1004003)] [gbkey=CDS]
METGTVKWFNAEKGFGFIEIEGGKDVFVHFSAIQGEGFKSLDEGQRVQFDIAQGNRGPQAENVVKL
>lcl|NZ_CP026520.1_prot_WP_042234204.1_862 [locus_tag=PC41400_RS04500] [protein=metallophosphoesterase] [protein_id=WP_042234204.1] [location=1004799..1005794] [gbkey=CDS]
MKNKAGLTLLLSASLLVLSTVPAGAVQERADKPVLTFPVISDVHVDAAKKDSQRKFTEALEDLNRVNPNADALVINGDLT
GGSARDYTRLKELMMSTPHPQKTFATIGNHEFYSAWLNAKGNWSPNSFPNKETEAASIGRFLSFTGEKKVYYEEEIKGYP
FIFLGSEQYRQTNPDNMEDAYLSKEQLNWLEQSLARHGQESGKPIFVFLHQPLPGTVAGTSVASNKRPIVQHEELRSILS
KYPQVIFFSGHTHWDLRQPNTLVRDKFTMVNSSSVERPWTSDGQGGEKLKDAKESEGLYVEVYEDRVSIQGRDFYAKKWI
PEAQFKVPVRK
>lcl|NZ_CP026520.1_prot_WP_241688790.1_863 [locus_tag=PC41400_RS30190] [protein=SMR family transporter] [protein_id=WP_241688790.1] [location=1005937..1006041] [gbkey=CDS]
MGYVYLAIAIIGEIFATSLLKSSEGSRGMPEPGR
>lcl|NZ_CP026520.1_prot_WP_042234201.1_864 [locus_tag=PC41400_RS04510] [protein=methyl-accepting chemotaxis protein] [protein_id=WP_042234201.1] [location=1006246..1008249] [gbkey=CDS]
MKPGLTLKFRDLSLRIKISLLLLVITSVPLIVATLILMSNSNGYISKQTEENMRIRAEMGAADIDLWMGQKINAVEGVVK
AHPEFATAASEEEVLPFLKNLQDSDPEMFYFAYIDRKDVSYGTEGGVTDVSAFQNVKNAKVSKKLAVTDILKDVGSKEDI
IIIDQPILDKNGEYAGIIQAIVSPKQIVSLVSRIHFGEKGYAFLLSGNRNILLHPDAADIGKPVKDIAPDLDNKLEASAD
QKGYLYADDGTVGEAAVAYSEVQKTGWKLFISAPKTEVYSTVSKAKTLAGIIIAVSVLLVILISILTSRAILRPLMNISG
MMRKAAAGDLTGRLDVKGRDEMGLLNEDINGMMDSFSSILDRMKNAIDHMASSSEELTAISAESAGVASRTSQTANSTAV
GAETQHDAAQQTAQAMEEMAIGIQKIAASSTQVAETSDSVRSEVNRGSETVQRTLNQMKAVGSNVRDSAGTIYSLQEKSE
EIQKIASTISDIARQTNLLALNASIEAARAGEHGRGFAVVAGEVKKLAEQTSHSTVEIQEILDEIVAATGRSTQSIESSL
EEVAQGEELMNTMEQSFRSIQTAILDVSAQIEEVSAATQEISAGTEEVSASSQEMVDISKQTLNQMRQVAQSMNGQLHSM
NDISSSAESLSSMAEEMQKLASAFTIK
>lcl|NZ_CP026520.1_prot_WP_042234199.1_865 [locus_tag=PC41400_RS04515] [protein=basic amino acid/polyamine antiporter] [protein_id=WP_042234199.1] [location=complement(1008524..1010002)] [gbkey=CDS]
MSNEPRKLGLLPLISLVVGSMIGGGIFSMPADMAQDAGSGAALIGWALTGIGVIMLALTFQSLANRLPSLDNGIVRYAET
GFGRFVGFISAWGYWLANMITIVASSVLLFGALDYFIPVFDGGNNVLAIVCASLLVWLLHSLQLSGIREATLVHIVTTMA
KLIPILVFIAALATVFHSPTFIADLWKTATETATGGSALGLSAIGSQVKQAMFITVWAFLGVEGAVVLSGRAKRKSDVGK
ATVIGLIGTLVLYMLVSLLSYGAMSRAALASLPTPSAAYVLEAAVGKWGAVLINAGLIVSLLGLLLGMSLLSAETLYVAG
KQKLMPAWLSGENKNGAPKGALWMTTGAVQLFLILVVTASSTYHALYLVATTAALMPYLLSSLFLLKTAIGGKAYAPQDT
TGESPLWNGPRKRELAYGAAATLYSLFLLYAAGLTNLLYVCMLYTLGLAFYSFSRREKGIRAMYRPAEAFAAAAIVVLGL
YSALQFISGHAG
>lcl|NZ_CP026520.1_prot_WP_042234197.1_866 [locus_tag=PC41400_RS04520] [protein=SDR family oxidoreductase] [protein_id=WP_042234197.1] [location=complement(1010143..1011042)] [gbkey=CDS]
MYPIYPYVGKETICKEQPIPFPAQHQPRQPGIESEMIPRPIAENPYYTGSGKLLDRVALITGGDSGIGRATAISFAKEGA
AVAIVFLYEWSDAEETRARIEQLGGRCLLIQADLRSKAASCEVVARTLSAFGRLDILVNNHGVQFPQQSILDISEEQLYD
TYRTNIFPFFFMTQAALPHLCCGSSIINTASITAYRGQKDLIDYSSTKGAVVSFTRSLALSLVDEGIRVNAVAPGPIWTP
LIPSSFSAEEVRTFGTEVPMKRAGQPFELAPAYVYLASDDSSYVTGQTIHVNGGGMITS
>lcl|NZ_CP026520.1_prot_WP_042234194.1_867 [locus_tag=PC41400_RS04525] [protein=NUDIX domain-containing protein] [protein_id=WP_042234194.1] [location=complement(1011115..1011777)] [gbkey=CDS]
MTPEMFDIYDEDANRIGTASREEVHTAGYWHQTFHCWLVRPEGNRRMVLFQLRSPSKDTFPDRFDITAAGHLTAGETIQE
ASRELQEELGLRVPFDRLSLLFTVRTDERGTVRGRTFIDREISHVFGYESDWPLDAYTLQEEEVSGLYEAELRELADLFE
GTIDSLEVRGVLAPSTSADGNDGLSQAGSDAESVRTVAKQDFVPHPDDYYLRVFELLGKI
>lcl|NZ_CP026520.1_prot_WP_042234192.1_868 [locus_tag=PC41400_RS04530] [protein=radical SAM protein] [protein_id=WP_042234192.1] [location=complement(1011917..1013203)] [gbkey=CDS]
MNPVRYEKITAKQVLNPVKTPAMPFEHSINPYRGCQHGCSFCYARAMHAFLGLNGDDTFQTNILMKENAPEALREQLRKA
ARSRKGLAGIGRIAIGTATDPYQQAEGRAKLTRGCLEVLTEYPVPLTITTRSPLILRDLDLLRKLPVYAVNISLNTLNRT
TWRNFEPGAPSPSKRLETVRALTDAGIPAGIFMAPMLPYITDGRDELEELIAAAAEHGAQFVMASYLRLSASDVKVWFFE
TIRRHYPQLVGKYAGLYASSAYAPREYRDPVRRYVDELLERYGLRDLDEEEEAPRPGERAAGAATPGRSSGKPLEIAWEP
PAGSPGSACGGCPAANAGAGAVASAGAAASASAVASAGAAADGGFAGAAGDSASRRGFADGSRTARQSAGTPGAAQAVAA
ADGTAAAEAGMPLALEPDSPPAQLVFTF
>lcl|NZ_CP026520.1_prot_WP_042234230.1_869 [locus_tag=PC41400_RS04540] [protein=MFS transporter] [protein_id=WP_042234230.1] [location=1013666..1014862] [gbkey=CDS]
MPEEKNNMSLPGFARKIIAATVFINMGRFSVFTFLAVYLTHTLQFPAWQSGTVLTVALLINQVVPLFSGMIGDRIGYSTM
LVAGTFLAALGYLGIYALDSFGGLVLAACFIGIGPAMYEPSIKSLFGNLPHVLRRRAFTYFNQALNAGAVLGAVTGGILV
SFHTSYPMLFGAGLFLLVSLLLLTQAKSFPKGRHSAKMLDSYKKVLRNRPFLAFSGAMAMFWIMFTQLSVSLPLEVYRMS
GNERLVSSVVIFNGLYAVLFMFLLRGVFQRNPAAVIVKYGMLIIGLGLLAVPWVPSVLWVLLCVLVFTSGETLVLPGVDI
AIAEYSMNEDTGAFYGAFGLSYAVGGTIGNYLGTWLSGEYGGTVWPWVIYGMIGLCGFICLQVLQALDDKTAKTVHTA
>lcl|NZ_CP026520.1_prot_WP_206735013.1_870 [locus_tag=PC41400_RS04545] [protein=cysteine hydrolase family protein] [protein_id=WP_206735013.1] [location=complement(1015182..1015736)] [gbkey=CDS]
MNDKAVLLVIDVQKAFDDPRWGRRNNPEAENNIALLLKAWRETGRPVIHVQHIGVTPQSLFRAGSAAVEPKPEALPMPYE
AVIRKSVNSAFIGTDLEARLRVLGCTEVVVAGLTTNHCVETTARMSGNLGFRTILAADAAATFDRTGPDGRVHAAEDIHA
MTLANLHGEFAEIADTRTLLEAVQ
>lcl|NZ_CP026520.1_prot_WP_042234228.1_871 [locus_tag=PC41400_RS04550] [protein=Lrp/AsnC family transcriptional regulator] [protein_id=WP_042234228.1] [location=1015896..1016339] [gbkey=CDS]
MDQTDRSILALLQENGRMSMTELGKQVGLTSPAAAERVRRLEDKGVLKGYRAVVSPEAVGKAVIAYILMETEHCRRYVEF
VRQVPEVAECHRIAGPYSYLTKLVTSSVRELETFIDASMEFGKPSTLVVLSSPLAQAPILPVKESVL
>lcl|NZ_CP026520.1_prot_WP_042234188.1_872 [gene=pyrH] [locus_tag=PC41400_RS04555] [protein=UMP kinase] [protein_id=WP_042234188.1] [location=1016550..1017287] [gbkey=CDS]
MKYRRVLIKLSGGAVAGSGDFGFDPERLEHIAREILSVLELGVEVSLVIGGGNIFRGNMGENWGIERAEADNIGTLATVI
NSLMLRGVLKARTNAEVRVMTAIPVTSVAEPYIRLRAIHHLEKGYIVIFAGGNGQPFVTTDYPSVQRAIEVNCDAILVAK
QGVDGVYDKDPNIYGDARMYESLHYNDVLQNDLRVMDQSAFILARDYGMPIHVFNFDKPGSMKKIVTGIHSGTTIHEAAK
LTFSV
>lcl|NZ_CP026520.1_prot_WP_042234186.1_873 [locus_tag=PC41400_RS04560] [protein=glutathione peroxidase] [protein_id=WP_042234186.1] [location=1017527..1018006] [gbkey=CDS]
MGVYEFTAQRINGEEQSLEDYKGKVLLIVNTASKCGFTPQYQGLQELYDAYKDQGLVVLGFPSNQFMEQEPGTNEEIEQF
CQVNYGVTFPMYSKIDVKGSDAHPLFQYLTRHTAGILSKEVKWNFSKFLVNRSGDVVSRYAPTTAPSKISGDIEKLLAE
>lcl|NZ_CP026520.1_prot_WP_042234185.1_874 [locus_tag=PC41400_RS04565] [protein=MarR family transcriptional regulator] [protein_id=WP_042234185.1] [location=1018030..1018485] [gbkey=CDS]
MNPPSEQKLENQLCFALYACSREITRMYHPMLEELGLTYPQYLVLLVLWEKDESTVKELGERLFLDSGTLTPMLKRMEAS
GLLKRTRSSRDERSVVITLTDKGRALEQPSTCIPAQLMARAGGTEEQVGDLLERVKQLLAAVNHANRNSEE
>lcl|NZ_CP026520.1_prot_WP_042234182.1_875 [locus_tag=PC41400_RS04570] [protein=S-layer homology domain-containing protein] [protein_id=WP_042234182.1] [location=complement(1018724..1020325)] [gbkey=CDS]
MKMRLLSLKIILAILLILPLFPAGPARAGLTSTPFPDVDSAKYSWAVQSITFMAQSGVLQGYPDGTFKPGRYVNKEEWAV
MVYRLFDKYRPSLRSDALNYSRPSYVDVPPSHWAYREISDTFVAVGDMTVTARSGKPVFNPEYAFTRWDLAMMLYGLYGI
DGGPESTNVCGDMNRFKDITTRLFNTENELESFRSKDNRYNSELPPGTIYPILFLKEADGTCGANYDDYSALAAGALIEM
NKTGILTADAAGKFYPSAKLTRAEAVTVLHRILDHLKTRGVLGNYSSINPTGILRPVPMPAPKPGEAAEGPGPFDRDLQR
DGEFNTIVYPNGKKYMRLELTSKDAADLYLYMNGDIAFLRQEELPRYIQVDRVSQIGIKSQSRNPEKRSPLAAYPATLKV
TFYDEIPSTGTGTGTGTGTGIGKPGKDNGGQTGGITAPPGAAAVQDLRSQGEINQTFAPNGSKYMNVNLVSQDKVDVYVQ
LGTGTAFLKQEELPAKVNVQGMSSVTVRTQSRNPEKRTSQQTYNAYLYVTFSN
>lcl|NZ_CP026520.1_prot_WP_042234180.1_876 [locus_tag=PC41400_RS04575] [protein=nucleobase:cation symporter-2 family protein] [protein_id=WP_042234180.1] [location=complement(1020674..1022008)] [gbkey=CDS]
MFNWFKTGSLGLQHVLAMYAGAVIVPIIVAGQLHLPPEQLSYLISIDLLTCGIATLLQVWGNRIFGIKLPVMLGCAFQAV
APMIAIGGQHGMTAIYGSIIASGLIIFVFGGLLGKLARFFPPVVTGSVVTIIGLTLIPVAIQDMGGGQNSADFGSLQNLA
LSFGVLLFILIMNRFFGGFIRAVSILLGLIFGTIAAAFMGKVSFQGVLDASWFHMVQPLYFGVPTFHLTSILTMTLVAIV
SVMESTGVFMALSKICDKDLTDRDLARGYRAEGLASIIGGLFNSFPYTTYSQNVGLVQLTKVKKRSVIAAAGVILILLGS
VPKIAALTMLIPTAVLGGATIAMFGMVVSSGIKLLSSVDLNQHENLLVVACSVGMGLGVTVAPTLFSHLPESVQILTGNG
IVAGSFTAIVLNLIFSIGKSRSKAPDSHSAPNQEKHEFSLKASS
>lcl|NZ_CP026520.1_prot_WP_042234179.1_877 [locus_tag=PC41400_RS04580] [protein=xanthine phosphoribosyltransferase] [protein_id=WP_042234179.1] [location=complement(1021992..1022603)] [gbkey=CDS]
MELLKRKIEDEGTVLSGQVLKVDAFLNHQIDPVLMLEIGKEFSRLFAGEGVTKVLTLESSGIAPSVMAALEMKVPVVFAR
KQKSLTLTDNLYVERVYSFTKQQTNEVTVAKKFLSADDRVLIIDDFLANGEAALGLANIVKAAGAEVAGIGIVIEKSFQP
GADKLKAAGYRVESLARVASLDNGKVTFLNNSVSEEDTHVQLV
>lcl|NZ_CP026520.1_prot_WP_042234177.1_878 [gene=bstA] [locus_tag=PC41400_RS04585] [protein=bacillithiol transferase BstA] [protein_id=WP_042234177.1] [location=1023119..1023649] [gbkey=CDS]
MGEDLRYPIGTFTYEGEISPEQIQDWLEELAAAPSRLRRAVAGLTEEQLDTPYRPDGWTVRQVIHHVADSHMNSYVRFKL
GATENEPVIKPYEEQLWAELEDGKTAPPEISLALLDALHDRWVRWLRTLDEAGWARTFLHPVNGPTTLARGLGLYVWHGK
HHTAHVTSLRERMSWH
>lcl|NZ_CP026520.1_prot_WP_042234176.1_879 [locus_tag=PC41400_RS04590] [protein=thiamine diphosphokinase] [protein_id=WP_042234176.1] [location=complement(1023973..1024632)] [gbkey=CDS]
MTLTQQRIFIVTGGAFGPWAEEHLNTLNPGTDLLVGADRGALHLVRSGYRPRLSLGDFDSVTPEELAEIRSGSEEFTACD
PVDKNWTDTELAYTWALERSPSEIILLGALGTRLDHTLANIHLLRKGAEAGIRSRIIDDHNDIELVRDRIVITAGPYAQV
SLLPLTTEVTGITLSGFQYPLENATLQIGQSLGISNVLQAGQGTISIRSGLLLVIRSRD
>lcl|NZ_CP026520.1_prot_WP_042234174.1_880 [locus_tag=PC41400_RS04595] [protein=hypothetical protein] [protein_id=WP_042234174.1] [location=complement(1024977..1025216)] [gbkey=CDS]
MSNDKTMDPMSHETVETDGIYADEAGREVTLKRGDKFPADLVLGETTWEMKGFSLNEAEIDHQQTENTPPRQHAKTGDR
>lcl|NZ_CP026520.1_prot_WP_042234172.1_881 [locus_tag=PC41400_RS04600] [protein=ATP-binding protein] [protein_id=WP_042234172.1] [location=1025452..1027302] [gbkey=CDS]
MTKFIQAIALGLVFTALFLFIGVKDVRAETSLPMRVNAPPVLLENWEYCWSEEEGAGRTEGPPHSSGDWKAADQLLNPQD
RMGRSHLWLRVPLPPNKAADPGLLIRAFDNVEVYTEQGLAYRFGDWQEGRKYRYYGTPRRIVPLPENTQGMLYVHLQSES
RNIGIEDGAMYGSESGFIKRMVRQEFFRAVLGVSYILAGIMAFCVFLGLRSHKVFLDFSLFSICFGIYSICRSSVVYLVF
DHPAILTYAELIALFGGVIGILMFVEHLLGSGRGAYIRRAWQLHLAYSTGTLLLHACRVVSVVEIVDWYQLLLVLSMPLV
IVRAAVCAMKGNRGAKLLLVGIGVICLTGLIDIFQNRLNIIIQFSPMTYLGVSVFILLLVVLLIRQFADIMLQLKRSEKL
SLVGQMSAGMAHELRNPLTIISGFLQLSRKQAPDAPYLAMMSSEVNRMNEIIDDFLLLSKPVNSRFDRHEVLPLLRETLQ
LFDNQRKEAGVSAILEEEGAIPKIDCDPNQLKQVFINMVKNAIEAMPEGGLLHITVSALPKNKVGIRFADQGKGIEPGHL
TRIGEPFFTTKENGTGLGLMVSNKIIENHAGILRVSSRLGLGTTIDIILPVHIKTA
>lcl|NZ_CP026520.1_prot_WP_042234170.1_882 [locus_tag=PC41400_RS04605] [protein=response regulator transcription factor] [protein_id=WP_042234170.1] [location=complement(1027415..1028113)] [gbkey=CDS]
MQASKILVVDDDHHILEIVSLYLRRDGFEVITADHGDHVLQLITEQRPDLILLDILLPGMDGIELCRQLRKTSDIPVIFL
SAKSEDIDIILGLSIGGDDYVTKPFSPAQLVARVKALLRRCSSSRNLADQHVLKFPDLEIDLHSHIVKVQERPIALSAKE
FDLLSLLAQNPNRVYRIEQLFEQVWSLESFGDPRTLIVHISNLRKKIEPNPSDPRYIITVRGVGYKFNGSPV
>lcl|NZ_CP026520.1_prot_WP_042234168.1_883 [locus_tag=PC41400_RS04610] [protein=glycosyltransferase family 2 protein] [protein_id=WP_042234168.1] [location=1028452..1029309] [gbkey=CDS]
MRESQSPKTPKEPLISAVVPTYNRPYVLGELLETLAAQNYPNLEVLVVNDAGVSVDFVAEAFPELAVTVINQPVNLKHVH
ARNEALRRARGELILLCDDDDLLTPGHIRRMLAALEETGADFAYSDVELFDYRVEGRTRIPVSRRLFAYEFDLAEMRKFS
TYVPSGTLYRRCIHDTLGEYDPEVHHYWDWDFFLRTAGSYKIVRVPVASVLYAFSLKGDNVSGNRQEGNPYLGRLCAKHG
LGELPSENFFTLLDKPHMRQREASSTIIWDGEPVVSRLAEQAESV
>lcl|NZ_CP026520.1_prot_WP_042234166.1_884 [locus_tag=PC41400_RS04615] [protein=glycosyltransferase] [protein_id=WP_042234166.1] [location=1029602..1030771] [gbkey=CDS]
MKKNGHKVLILSGSYGDGHKQAAEAIRREIAQRYPGAETVLLDFMEWTHPFMNSIGRYMFLKGLQTFPSAYGYVYNKTRE
ANLFSYILKQFNRFGLGRMMKLLREVEPTVVVSTFPPAAGAMSAVRAYGLYDVPTATVITDHTDHSYWVYPETDKYLVGS
DTVRDGLIQAGVPAGRIEVTGIPIRPEFQGTYESAQTRRLLGLDPELPTILFMGGGCGMMGGELASKLEAFEALPVRTQL
VVVCGNNEKLRAQITELASHSRHRILATGYVNNVHEWMAAADLLVTKPGGLTISEAMAMKLPIVIYKSLPGQEEDNARFL
LQAGIAVKAKNMNDLIGQLGILLRSPSLLYTMRENLGRTRRSNASAEAVRIVLETRSERRYAAALQEIW
>lcl|NZ_CP026520.1_prot_WP_053228782.1_885 [locus_tag=PC41400_RS04620] [protein=glycosyltransferase] [protein_id=WP_053228782.1] [location=1030836..1031984] [gbkey=CDS]
MPMNPDAKILILYASYGDGHLQVSRALRQSFETQGVSNVELVDLFAQAHPWFNSFTRFVYLKTSSIGLYGLSYYLTQEMR
ADALLSRWLHSLGDRKLRQMIREHQPAAVVNTFPLKGMPDSGPHSGETIPTFSIVTDYDLHSRWIHPDVQHYFVPSEELK
TLLVAKGVPADNIKVSGIPLRPSFQAAGDGNAALTSGEAEQAAPFVLMMAGAYGVLRGLRGLCAKLLRDVPDVRIVLVCG
RNERLKRYFETAFAGQERVCVHGFVEDIHRLMAASACMITKAGGVTLSEALALDVPVIVYRPFLGQEKENADYFVKKGAA
LAAFNPDELVRQVGRVIRGEWPPESARTSARNLHKKNAGQTIVSHILEEIGHHPARSSEQAR
>lcl|NZ_CP026520.1_prot_WP_042234165.1_886 [locus_tag=PC41400_RS04625] [protein=MFS transporter] [protein_id=WP_042234165.1] [location=1031990..1033204] [gbkey=CDS]
MDESKQRRLSARLIGYAFILFVVEFVRGAYLLSYLPVYAANVLHYPVTVVGLAISIHYVVDMIVKGLAGLLLDRFPARVI
VGGGMFFSLMGLLLLQGPHNVFIFLLSAALFGVGISPIWLVCLSRITAEQRAFQMGVLYTVWLTGLGAGPVVINFVLDYS
LSVTYFILLGLWGLGCLLSMGMGGRAEGTRRKTVTWKVQAGVLKKQLGAAKPLLPPMMLQTLAAGLLVPVLPGFASGVMG
LQYAEYSMVLLAGGLCTVICLMPMGKLSDRFGLRGFLILGFAFLAAALYVMTFIKELEFALLTAALLGVSYACVLPAWNA
LMALYVPKGHQEMGWGTLSSVEGLGAFIGPVLGGWLADRYSEVFVIASSGVLLGAIALWYILVPLQPSDKTDGKAAVKGV
TYRG
>lcl|NZ_CP026520.1_prot_WP_241688791.1_887 [locus_tag=PC41400_RS04630] [protein=polysaccharide deacetylase family protein] [protein_id=WP_241688791.1] [location=1033197..1034843] [gbkey=CDS]
MVEGIIWVVFIFYLIYTLVPTFLIRRLGYGVFKSRSDSPCAALTFDDGPDPQYTPQLLDLLARKGTRATFFVLGAKAEHY
PDLIRRMHEEGHLVGLHNYMHHSNALMTPWKVRRTLEHSADVIERITGIRPVHYRPPWGVFNVFDFLLIRKFRIVLWSMA
AFDWRSRGGTDKIKRKLIGKVKDGAVILLHDSGDTLGANPDAPGYMIEALADVLEDARRRGIRFARIDEHLPGRRLPERL
VRRSAEKRTALPAASSPRDRASQAAGGAAAAAHGPRSADRAAATARDPQAAVPGPRTTGEAVSAQLPQPVMEAAPAREAP
QSEPQPPGKHRGLALRAWMLWERLFHRLYRIQPGDSEHRFLHYRIRAYHGQELVLEDGEIIRPGDKVAELHLDNEWLRQA
GSQNSSMMKLAVQMIRAMDKSMPRLAHRLLTDPTLADLKGVYGISMIHRGTKQFGFTVLPLHNRLWRWSSTLYLRVLLGM
LHPGGFGRLQENTARLNPRIIAMSTSQMKKRYAPGLSAAEPSSGAYYPEEAYGAVLQELPDGVAGKGG
>lcl|NZ_CP026520.1_prot_WP_174446849.1_888 [locus_tag=PC41400_RS04635] [protein=PRD domain-containing protein] [protein_id=WP_174446849.1] [location=1034951..1035832] [gbkey=CDS]
MKNSGIEVVKVLNNNVIIAAHPDYREVVVIGKGIGFNRKAKDRIPSEAVEKLFILTNEKEKEQYQQLIPQIDEKLIEVMN
EVILYISEHAGAPLNEHIHIALTDHIAFAIKRLEQGIVLQNPFLLETRELYPKEYAIASYVIRTINDKMNVHLPPAEVGF
VTLHIHGAITNRPVSEVNEYSRLIGELIGLIERQMHITIDRSTIAYSRLLRHLRFAIERTKRKETVHEPDKLGELLKQQY
PEIYSIAWTLMKVMQRRLRQPVYEAEAVYLTMHLIRLMQKDDSLEKEKPAGDS
>lcl|NZ_CP026520.1_prot_WP_042234161.1_889 [locus_tag=PC41400_RS04640] [protein=ABC transporter ATP-binding protein] [protein_id=WP_042234161.1] [location=1036139..1036915] [gbkey=CDS]
MGNLLLEVGRLNKAFETKKGLVHALHDINLDIKEGEFITVIGPSGCGKSTLLKIVAGLDTDHNGNVRLGGEEVTGPGIDK
GFIFQEPRLFPWLTVEKNIAADLSLKDPQVRAKVDNLIELVKLQGFEKSYPRELSGGMAQRVSIARALLRNPKILLLDEP
FGALDAFTRAHMQEVLLDIWQANRTTMVFVTHDIDEAVFLANRVVILKPRPGAIRNIVPVDLPFPRKKSSTSFQEIRYKI
LNEFEKLDELELVESAGI
>lcl|NZ_CP026520.1_prot_WP_042234158.1_890 [locus_tag=PC41400_RS04645] [protein=ABC transporter permease] [protein_id=WP_042234158.1] [location=1036932..1037804] [gbkey=CDS]
MSKPSNLLQGTGATLLGASPDSVPAEPAPAGSSKRRRRLESFLLGAIVPAAALILWQVLGDAGYISRLLFPTPWTIAKSF
RSLLESGALAENVGISMLRAAAGFALGGGLGLLLGMLVGFFRRTEKVLDPSVQMIRMVPHLAIAPLFILWFGFGEPSKVL
LIAKGAFFPLYINTFLGIRGADKKLFEVTRVLGFSTYQQLARLIVPSALPNIFLGLRVSLGIAWLGLVVAELMGASSGLG
YLMSEARQFNKTAVVYVGIILFAVIGKAADSGVRLLERRLLQWQDSYGGK
>lcl|NZ_CP026520.1_prot_WP_042234157.1_891 [locus_tag=PC41400_RS04650] [protein=ABC transporter permease] [protein_id=WP_042234157.1] [location=1037820..1038671] [gbkey=CDS]
MNGGGEALRRLTGTHEQPGSLRKYGQTLLKAKTGDFALGAVLPVLVLIVWQAAGDFGWISTFFLPTPLMIGAALRELAIS
GELTEHVGISLWRALTGFLLGGSLGLLFGLLTGYFRKAEQLLDPTLQMIRLVPHLAAAPLIILWFGFGETSKTLIIANGA
FFPLYVHTYLGIRSVENKLFDVAKVLRFSPYKQLLRLIIPASLPNILLGLRLSLAVSWLGLVVAELIGSQSGIGFLINFA
KQNSATEIIFVGVILFAVVGKLVDTFVRLLEAKLLRWRDSFEG
>lcl|NZ_CP026520.1_prot_WP_042234220.1_892 [locus_tag=PC41400_RS04655] [protein=NrtA/SsuA/CpmA family ABC transporter substrate-binding protein] [protein_id=WP_042234220.1] [location=1038753..1039769] [gbkey=CDS]
MTHTAYTKKYTHLIGFVLLLAALVLLASACGRAEADSSGGAGPGGKAPSVLRVGYIGANKLNLPSGAEGWGFYKGGAKEE
LKKLGITDVQFVGFPNGPDLSESLISGRIDIGSLGDTPAILARSAGAPSKVITLNSSGSNAFILGKKNGPTTLADLKGKK
ISVKKGSYMHRYLAGLLKEQGIKDVQLIHLLDEDADAALVRGEVDAIATTEVRGLKMIPQGFPVLDEAAKDHPALVGAGV
TVVSESYLKKVPDIAKVWNDLRIKALDDLKTQEDAYYAFVVENTGTPLDLVKKTSAVSRIQRESFTDSGIQALKATKDFL
VQDKLADTDFNVDDWLVK
>lcl|NZ_CP026520.1_prot_WP_042234155.1_893 [locus_tag=PC41400_RS04660] [protein=FAD-binding protein] [protein_id=WP_042234155.1] [location=1039846..1041543] [gbkey=CDS]
MSQKQPLQLTADVLVIGGGPAGAWAAWSAASRGARVVVADKGFLGSSGATAPGGTNLLYLPPDPALREEAVRKRMKGGGF
LSEPSWIHGVLDQVYESLELVESWGYPFRRDENGVPMRDHLQGPEYMKLMRRAVIKAGARILDQSPALELLVDEHGVGGA
RGVSRLDDREWEVRAGAVVIATGGCAFLSKGLGCNVLTGDGLLMAAELGVELSGMEFSRQYAPSAAFATNTRGRLLNWAT
FYAEDGTALHSEGPRPHDFLPKMLSKGPVYAVMDKADTPEKRAILRSSHAIFFLPLDRAGIDPFTQPFPLTLRYEGTVRG
TGGIRLVGEDCSTTVAGLYAAGDAATRERVTGAQSGGGAFNASWAISSGTWSGRAAARYALALGGRAGQRSLRPAGGAGL
AGARGTSAMTDVPGIIGAVQREVFPLEINYFRSEPVLRESLQRLHGLWPLVQGGIPGGDARERVRAREAAAMVAAARWIH
TAGLARRETRGMHTLAEYPHTDPAYGRRLLVSGLDDIHVKFEGGGEVPPRPDPAAAAHLGLPRQRGHEPLQPTGGDAVRK
EVTPV
>lcl|NZ_CP026520.1_prot_WP_042234153.1_894 [locus_tag=PC41400_RS04665] [protein=ferredoxin family protein] [protein_id=WP_042234153.1] [location=1041540..1041941] [gbkey=CDS]
MIELVSTARCVGCSMCVKVCPTNVFDEGPDRIPVIARQNDCQTCFICEVYCPVDALYVSPYGDEAEIVDERELAESGVLG
SWREKIGWGPGRVKLAKLDTTPFIDRVLPLQPPLDEEETAFILTPDYPRKPAG
>lcl|NZ_CP026520.1_prot_WP_042234151.1_895 [gene=ptsG] [locus_tag=PC41400_RS04670] [protein=glucose-specific PTS transporter subunit IIBC] [protein_id=WP_042234151.1] [location=1042563..1044623] [gbkey=CDS]
MLKGVFGALQQVGKALMLPVSILPAAGLLLAFGSAMQNPDTIELMPFLSGNGFQVAAHLMSSAGDIIFANLALLFAIGVA
IGLAGGDGVAGIAATIGYLIMNVTMGVLAGVTPFSLTLKDPSYASVLGIPTLQTGVFGGIIMGVIAALLYKRYHKIELPS
YLGFFAGKRFVPIITSLAALLTGLILIFIWPPVQQGLSTFSRSVLDANPVLASFLFGVGVRSLIPFGLHHIWNAPFWYEF
GEYIDKSGVLVRGDMKIFFAQLKDGVPLTAGTFMSGNFPFMMFGLPAACLAIYHTARPENKKLVAGIMGSAALTAFLTGI
TEPIEFSFLFVAPVLYVFHALLAGLSFAIMTLLDVKLGFTFSGGSIDFLLLGVLPNRTPWWLAILVGLVFAVIYYVVFRF
AITKFNLRTPGREEVQTEADTDDAPVSGASNAAGTGDLPSSILAALGGAPNISNLDACITRLRVQVKDKGVVDKERLKKL
GASGVLEVGNNVQAIFGTRSDSLKNQIQDVMAGRIPSTASVPVVESPDTGGSELMKEEAIVAPADGELMDITLVPDEVFS
QKMTGDGFAILPKNGTIVSPVNGKIFNVFPSKHAIGIQSDAGKELLVHIGVNTVKLKGQGFEVLVSEGDIVEAGQPIMKV
DLDYVKEHAPSIISPVIFTNLPEGTEIRLNKQGDVKSGDKNIITIK
>lcl|NZ_CP026520.1_prot_WP_009674656.1_896 [locus_tag=PC41400_RS04675] [protein=phosphocarrier protein HPr] [protein_id=WP_009674656.1] [location=1044720..1044992] [gbkey=CDS]
MQKTFKVTSDTGIHARPATALVNTSNKFKADVFAESKGKKVTMKSILGVLSLGLEENDVITFSAEGEDAAEAIAALEAIM
TKEGLGELHG
>lcl|NZ_CP026520.1_prot_WP_042234149.1_897 [gene=ptsP] [locus_tag=PC41400_RS04680] [protein=phosphoenolpyruvate--protein phosphotransferase] [protein_id=WP_042234149.1] [location=1044985..1046703] [gbkey=CDS]
MANIRGIAASAGIAIARAFRLEHPDLTVVRKTVEDTEAEVAKFEQALEKSSAELEAIKEQTLREMGEEKAEIFESHLLVL
GDPELVDPVKTKIRDEKVNAEFAMNEVAMTFVEMFENMNSDYLKERAADMRDVTKRVLAHLLGVKFVSPSSISEETVVVA
EDLTPSDTAQLNRTYVKGFTTNIGGRTSHSAIMARSLEIPAVVGTKNVLDLVKDGDLLIVDGLDGDVLINPEPAVVEEYR
AKQQRYAEQRAEWAKLRNEPTVTPDGHHVELAANIGTPADVAGVLENGGEAVGLYRTEFLYMGRDQLPTEEEQYKAYKAV
LEKMEGKPVVVRTLDIGGDKELPYLQLPKEMNPFLGFRAIRLCLEREDIFRTQLRALLRASVHGSLRIMFPMIATLDEFR
QAKALLLEEKAKLAAEGTPASDEIQLGIMVEIPSTAVMADQFAKEVDFFSIGTNDLIQYTMAADRMNERVSYLYQPYNPA
ILRLVKMVIDAAHREGKWAGMCGEMAGDTTAIPLLLGLGLDEFSMSATSILPARSQIVKLSLSEMRQLAARAIELSTAEE
VVALVNGHQPRA
>lcl|NZ_CP026520.1_prot_WP_042234147.1_898 [locus_tag=PC41400_RS04685] [protein=MmcQ/YjbR family DNA-binding protein] [protein_id=WP_042234147.1] [location=1046924..1047313] [gbkey=CDS]
MNTTEIMEYALSKPGAAADRPFDNETDVLRVGGKMFALLPSGAAPGEGSVTLKCPPEEAELLRMQYPEAVKPGYHMNKRH
WNTVRLDGSLDPQEIMDMVDVSYMLVRGSLSKKMQQHILQHGGSDWNES
>lcl|NZ_CP026520.1_prot_WP_231392149.1_899 [locus_tag=PC41400_RS04690] [protein=YitT family protein] [protein_id=WP_231392149.1] [location=1047726..1048553] [gbkey=CDS]
MQRIVFISLGAALFSIGLEVFLVPNNIIDGGITGISIMLSHFFHLPLGIFLTLLNLPFLFLGYKQIGKTFAISTLFGVIA
MSVGTSLLHPVPAVTIDPLLAAVFGGVILGVGIGLVIRSGGSLDGTEIVAILINKKSPFSVGEIVLFINLFILGSAGFVF
GWDHAMYSLIAYYIAMKMIDVTIEGLDQSKSVWIISEKYKEIGDALTQRLGRGVTYLHGEGAYTGDDKKVIFTVITRLEE
AKLKSIVEEWDENAFIAVGNIHDVKGGQFKKKDIH
>lcl|NZ_CP026520.1_prot_WP_042234145.1_900 [locus_tag=PC41400_RS04695] [protein=CstA-like transporter-associated (seleno)protein] [protein_id=WP_042234145.1] [location=complement(1048660..1048863)] [gbkey=CDS]
MFKRLKLIWSYRNQFIHLMVGVPHYETYVAHMKSAHKDQPVLSRAEFFRQAQDNRYDAKDGKISRCC
>lcl|NZ_CP026520.1_prot_WP_042234143.1_901 [locus_tag=PC41400_RS04700] [protein=carbon starvation CstA family protein] [protein_id=WP_042234143.1] [location=complement(1048856..1050946)] [gbkey=CDS]
MKIGWKSILLWGVISIVGAFAFGLVALNRGETINAAWLVITAVSCYAVAYRFYSRFLARKVFVLDDNRKTPAELNNDGKD
YVPTNKWVLFGHHFAAIAGAGPLVGPVLAAQMGYLPGTLWIILGVILGGAVQDFVILFASTRRNGKSLGDMIKDEIGKVT
GTIAMLGILGIMIILLAVLGLVVVKALIGSPWGMFTIAATIPIALLMGVYMRYIRPGRVGEASLIGVVLLILSLIGGKFV
AESAALAPLFSFSGETIALMMIAYGFFASVMPVWLLLAPRDYLSTFLKVGTIIGLALGIIFVLPDLSMPATTRFIDGTGP
VFAGNLFPFLFITIACGSVSGFHALISSGTTPKMLERESHARPIGYGAMLVESFVAIMALIAACIITPGTYFAINSSAAA
LGGTDVATAAATITNWGFTITPEQLTTLSKDIGESTILSRTGGAPTFAIGMAQILSGVIGGKTMMAFWYHFAILFEALFI
LTTIDAGTRVGRFLIQDLLGYASKSLGDTRSTVGNLIATALCVAGWGYFLYQGVVDPLGGINSLWALFGIANQMLAGIAL
VLGTTILFKMGKKAYAWVTLLPTTWLLIVTMSAGWQKLFHEKANIGFLSHAKQFQSALDSGKIVAPAKTAAQMKQIVMND
YIDAALCGFFMIVVLIVFISAIRVWLKAISNKPLDLYEAPYIPRSGQGRSEATKYV
>lcl|NZ_CP026520.1_prot_WP_042234141.1_902 [locus_tag=PC41400_RS04705] [protein=hypothetical protein] [protein_id=WP_042234141.1] [location=1051809..1052249] [gbkey=CDS]
MDERKIQTYTLVSAPNVTGENFIKLLNIYRRPFVVLVNNPAEGQRMKKLGVRHIVEVDTVHHEAWEVPGLPIGDVYLFED
SLPLCCRYLQMVRSWTTGTIFVVKTNASPRLIYKALGADHVVHTNTRQVSFLLGAAPGQVPEEVLR
>lcl|NZ_CP026520.1_prot_WP_042234139.1_903 [gene=kdpA] [locus_tag=PC41400_RS04715] [protein=potassium-transporting ATPase subunit KdpA] [protein_id=WP_042234139.1] [location=1052575..1054254] [gbkey=CDS]
MEIMQIVIVIAALCLLVKPLGTYMYHVFSNEKNRTDKWFGWLERPIYKLAGIRSENGMTWKKYAGSLVLTNIVLVAVGYL
ILRLQRALPLNPNGIDNMEATLSFNTIISFMTNTNLQHYSGETGLSYFSQMAVIMMMMFTSAVTGIVVAIAFIRGLTSKG
STIGNFFADFVKGHTRLLLPLAIVVTLVLVGLKVPQTLDPTVAAKTLEGGEQQIAIGPVASLVSIKHLGTNGGGFFGVNS
SHPFENPSPLTNVIEMLSMWCIPASLTYTFGRFAKNRKQGWVIFRAMMTLFLVFLSVAFFSEKAGNPALNALGADSSIGS
MEGKEVRFGIAQSALFTTVTTAATTGTVNNMHDTLTPLGGMVPLAQMMLNCVFGGDGVGLVNMLMYAILAVFLSGLMVGR
TPEFLGRKIESREMKLIAVAILAHPLIILAPTAIAFMTELGKGAVTNPGFHGISQVLYEFTSSAANNGSGFEGLADNTPF
WNISTGIVMLLGRYVSIIALLGVAGSLGRKQWVPETIGTFRTDNALFTGIVVGTVVIIGALTFLPVIALGPVAEYLTLR
>lcl|NZ_CP026520.1_prot_WP_042234136.1_904 [gene=kdpB] [locus_tag=PC41400_RS04720] [protein=potassium-transporting ATPase subunit KdpB] [protein_id=WP_042234136.1] [location=1054275..1056320] [gbkey=CDS]
MSSTASQTKRKSMLSGDIVKQAIVQSFLKLNPVTMMKNPVMFVVEIGTFIVLLMTVLPGYFGTQDRIGFNLTVFLILLFT
VLFANFAEALAEGRGKAQADSLKKSKQEIQANKVVGSGIKVVPSTELRKGDIVTVAQGEMIPGDGEVIEGLASVDESAIT
GESAPVIKEAGGDFSSVTGGTRVVSDAIRVRITSDPGESFIDRMISLVEGASRQKTPNEIALNTLLITLTLIFLIVVVTL
KPIAGFVNVELDTAVLIALLVCLIPTTIGALLSAIGIAGMDRVTQFNVLAMSGKAVEAAGDINTIILDKTGTITFGNRMA
SEFIPVGGASTGEAATWAAVSSVKDETPEGRSVLELMKKQNLVYEASLAEGGEFIDFKAETRMSGIDLKDGRKVRKGAVD
AVVRWVQEQGGTVPADLKANGDAIATEGGTPLAVAVDGKIYGLIYLKDTVKPGMKERFDTLRQMGIKTIMCTGDNPLTAA
TIAKEAGVDGFIAESKPEDKIAVIRKEQAEGKLVAMTGDGTNDAPALAQADVGLAMNSGTIAAKEAANMVDLDSDPSKII
EVVGIGKQLLMTRGALTTFSIANDIAKYFAIIPAMFMLAIPQMEALNIMKLGSPLSAVLSALIFNAVIIPLLIPLAMKGV
KYRPMSSTKLLSRNLFVYGLGGVVVPFIGIKIIDLVVHVWI
>lcl|NZ_CP026520.1_prot_WP_042234134.1_905 [gene=kdpC] [locus_tag=PC41400_RS04725] [protein=potassium-transporting ATPase subunit KdpC] [protein_id=WP_042234134.1] [location=1056456..1057073] [gbkey=CDS]
MNPGASVPKENGEWNGSVLSVAVRISILLIVLCGIIYPLVSTGLAQVIFPKQANGSVIKNEQGEIVGSELIGQLFADPKY
FHGRVSSIEYKAEASGSNNYAPSNPELVKRTQESMDTWKKENPDVPVNQVPISLLTNSASGLDPHITPESAQVQIPRIAK
VTGISTAQLQELVKKNTDGRDLGIFGEPRVNVLELNLDVKKLAGK
>lcl|NZ_CP026520.1_prot_WP_042234132.1_906 [locus_tag=PC41400_RS04730] [protein=histidine kinase] [protein_id=WP_042234132.1] [location=1057402..1059756] [gbkey=CDS]
MDNFRRKTPEEILLSISDIHRGRLKIYIGAVSGSGKTYHMLREGQTLREQGIDVVLCAISMEGRPETMEQAGGLERVESI
EWRKDGLRQQDLPLDDLVARNPEVVLTDGLAHRNRPGALYPTRLEDIKFLLNKGISVLATVNVYELEGVRELARKMTGIV
AEHTVPTDTLELADEVRLIDATPETILNRLEEGGMKGCQWHERKVLERGNIGVLRELALRLVAEGVNESLEKHRGEMGMT
GPSGADERILVSTQYHWNGSIYVRRGQQIADRLSGDLFVVSFTVPGKALSKDAATFRRSIIKLSEKVGGTFDELPFPGRR
KLAQALIRYGIDRKVTRIVLGHSKHSRWQEFWQGSVLNDAVSRLQGIDLFVVADRAERDGERILPVKRKSDARQPVELYR
RPSPQEMERQIRKARRGRFKVYIGAAPGVGKTYSMLREGNDLLRKNIDVVIGLLELHGRKETADQVGDLEQIPRKIRTYR
GTELQEMDTEAVIRRMPDLVLVDELAHTNVPQSVNKKRYEDILEILNAGISVISTVNVQHLESLNDAVEQLTGVRVRETV
PDSILRLADEVELIDVTPQTLRQRMKEGKIYGMDKVDQALGHFFKTVNLIALRELALREIADDVDERLEAWERVGSLRGP
WRREEVIFVCVKLNQHAERIIRRGFRIAYRLKARWHVAYIRCGSAAGDEEARIAALQTLTERLGGRFEVLGAAGRRRTAD
GLLARALELSATQLIIGQARESLWERLFHGSIVKRIMRGARSMDVLVVAESESAKARRGLRTRG
>lcl|NZ_CP026520.1_prot_WP_009674680.1_907 [locus_tag=PC41400_RS04735] [protein=ABC transporter ATP-binding protein] [protein_id=WP_009674680.1] [location=1060121..1060960] [gbkey=CDS]
MIQIKHVRKTFKQRSSGSFTAIDDVSLTIRKGEFVSLLGPSGCGKSTVLNMVAGLEGYDEGTIEVNGTKVTGPGSDRVVV
FQEHALFPWLTVLDNVAFGLKQKGIPKKERYEKAMEQIKTVHLSKFADRYPHELSGGMKQRAAIARALAMDPEILLMDEP
FAALDEQTRLILHKELEEIWMKTGKTILFITHNIREAVILSDRVVVMSTRPGTIKKEFAVQAARPRDNADSVIHHVESAI
MDVLAEELEKVVKEEMGDEYSLKKNDFSRDSADSVGGGI
>lcl|NZ_CP026520.1_prot_WP_042234130.1_908 [locus_tag=PC41400_RS04740] [protein=ABC transporter permease] [protein_id=WP_042234130.1] [location=1060893..1061657] [gbkey=CDS]
MSTALRRTIFLVILLIAWEAGYRIFDWGWKFPSLFQTLDAFRLGFTDGQLVEATLRSLGRLLTGFCISLVVGTLLGFLFA
RFRFLDDTLGFVVVALQTVPSIAWLPFAIIWFGLTDTSVIFITALGATWTMAMSSRTGIMNIPPIYLKAAQTMGTGNGLR
MFWQIMVPAAFPHLINGVRVAWAFAWRALVAGELIAKGVGLGQLLQQGRDIADTALMLCIVIIIAVLGTISDHLCFKKLE
DKILLRYGLAISGK
>lcl|NZ_CP026520.1_prot_WP_042234128.1_909 [locus_tag=PC41400_RS04745] [protein=ABC transporter substrate-binding protein] [protein_id=WP_042234128.1] [location=1061694..1062743] [gbkey=CDS]
MKKTAWLLALILLVGTVLTACGGGDSKNSGGNGDGKKPETVRVGIFKNVTHAAGYVALENKYFEKYWGEGVKVEVTAFDN
GSDFSIALATDQIDLGFVGPGPSTNQYLKSKNFKVISGSNNGGAVLAVRKDAGISSVKDLVGKTVAIPTRGSTNEISLRL
LLQQEGLKVTTDKSGVEIIARAPADTLVAMRQKEVDATLIPEPWGTQMEKEGIGQILVDWDKIPPNNGNYPLTILVAGDK
FLKDHRDMAKGAIQANIDAINFIKSNPDKSYDLINNQLKKLSGKGLDNELIKAALSRLNLTPDVSKEAIEEMAKVSIDAG
FIKNVKKEELDLSKFLDLSLLDEVKKEKK
>lcl|NZ_CP026520.1_prot_WP_042234218.1_910 [locus_tag=PC41400_RS04750] [protein=FAD-dependent oxidoreductase] [protein_id=WP_042234218.1] [location=complement(1062910..1064259)] [gbkey=CDS]
MNDSQFDTLIVGSGLAGLSCAFELAEQGQRVHIIEAAPFAGGRTSNWSDAGMEVESGFHKFIGFYEALPNLLRRARIKPN
SMLTWEKTFEIRLPDGETAGEFGIAPLKSPIKTITGAFTNNDIISPADKASLMPFVAEGLKLLSENPMELDAYSVAEFAR
KHGVKDEALRHVLGPLTSGVLFLPPEDYSALVFFGLFAPGVPKFYKMQIGGFNGGMTDVMITPLLAALEKRGCRISLNTP
AEGLSLEDDRVTGVSLQEGGTLRAANVVIATEISAAKRLLQPPFGGEAWYQPLGELPTMPDVTIQLELSEPLLPQDRVTF
GPGTCIGSFSEQSRTTFPHVPGRVSMILSPPDEFIDMPDDLVFERVCADADKLGLDLRAKAKDYRVIRRPDHFYSVRPGS
EKFRPEQRTPVPGLALAGDYTRQPMFATMEGAVLSGRKAAEAVLGRELS
>lcl|NZ_CP026520.1_prot_WP_042234126.1_911 [locus_tag=PC41400_RS04755] [protein=DUF2935 domain-containing protein] [protein_id=WP_042234126.1] [location=complement(1064344..1065222)] [gbkey=CDS]
MVPTPKLSVWTEHRFWLGILYDHATFVWQALAPSEQPFIQTANGFRQAFAALIKQVELTPQELPVSSPVMIKLAQDILPA
ASGYYQFEGHIQSLRISNKIVLELTPSYFNGTLIENEEYLRLLSFWTRGVEAEELTLYGLLDMWLEDQLGHAVLLWDHLD
PVEIELSEQARRYSNTFQAYMVKNRAMGGYLRFTPPGFPAQRRFAADVAVAIIKFYDFVVQVIDMYVDEELLSRLTLRFL
DHHLPESCYFLNKLAVYEPNLKIPGNCFLYKPIAPNENIPYPQLTPPQPRED
>lcl|NZ_CP026520.1_prot_WP_042234124.1_912 [locus_tag=PC41400_RS04760] [protein=polymer-forming cytoskeletal protein] [protein_id=WP_042234124.1] [location=complement(1065307..1065747)] [gbkey=CDS]
MVFDKKRNTDNKTTDTIIGESTVCEGKIISEASLRIEGQVLGDIECSGDIVIGEKAVVHSSIQARDVTIAGKVKGDVATK
GKLTVMSTGELIGNIQVRAFIMQEGGVFQGTSTMTASGQDDKAPSPISSAPSAKNRKEEQSLASAN
>lcl|NZ_CP026520.1_prot_WP_042234121.1_913 [locus_tag=PC41400_RS04765] [protein=M23 family metallopeptidase] [protein_id=WP_042234121.1] [location=complement(1065749..1066732)] [gbkey=CDS]
MKTKEDKKELTLMFIPGADKRVIRFKIPRISLYAVPSAAALVILGFSLTIYVQNTSHRSTTEQLTETFTGQEKQLTEQLM
LKDSELEKLQIELIDLSKQAEQFKSKLEEIRKLENKISPGTAVSSGKSTAPPPSSLPDVGGTEMPVGTREMNELAAQTKS
NLSGMIGDIDGLLVQLTESERKLEEARRLKRITPTIWPSDSRTVTSGFGVRQDPFNGRPNVHTGLDIDGELNDPVYAAAD
GTVTAAGYDSQHGNHIRIDHTRGIETEYLHLNKMEVQTGSTVRKGQIIGRVGTTGRSTGTHLHYEVHKNGTKINPSPYLI
SDRKDDN
>lcl|NZ_CP026520.1_prot_WP_042234119.1_914 [locus_tag=PC41400_RS04770] [protein=glycosyltransferase] [protein_id=WP_042234119.1] [location=complement(1066983..1068089)] [gbkey=CDS]
MRKKRILLLSEGFGSGHTQAAHALASALRRTNPGLIIRVLELGAFLHPTIAPWIFTAYRKTVVSQPKLYGKLYRSQSKKP
LNGFLQHALHRIFYRKAEAVIEQLRPDIIICTHPFPNLIVSRLKKQHGLPIPLCTLLTDYDAHGSWINDEVDRYFVSAPD
VKRRLVAKGVSSRRIKVTGLPVHPSISEQYSVDEIYLEFGLRDMPTVMLMGGGWGLLDEKLLKRIVQWRDRVQLLFCLGT
NEKARQQLQSNELYRHPNIHLFGYTREIGKLMEVSDLLITKPGGMTCTEAMIKGVPMLFYNPIPGQEEENCQYFTEHGYG
EELTSLSELDGYFTRFMENRRDKNKIARKGSSRRLSSLFAKSVMQMLR
>lcl|NZ_CP026520.1_prot_WP_042234117.1_915 [locus_tag=PC41400_RS04775] [protein=thymidine kinase] [protein_id=WP_042234117.1] [location=complement(1068196..1068774)] [gbkey=CDS]
MAKLYFRYGTMNSGKSIEVLRTVHNYEEQGKKVLLFTPSVDDRSGVGIVASRIGMQKSAIVIQDGVNMKELAAKERPNCI
LIDEAQFLNAGQIEQLAEIADDLNIPVIAYGLRADFMGQLFEGSASLFAIADKIEEIKTVCWYCDKKATMNMRCKDGVPI
FHGEQIQIGGNESYIPVCRKCYTDKKREAGQH
>lcl|NZ_CP026520.1_prot_WP_042234115.1_916 [locus_tag=PC41400_RS04780] [protein=YdcF family protein] [protein_id=WP_042234115.1] [location=1069021..1069650] [gbkey=CDS]
MKAVREGRGKNKQWKRWLYGVLSCMGALLLYIVGMSGWIYYVGENSRGTESDTIIILGAAVWNGKPSNAMKERLDTAVEA
YRGGLAKHIIATGGTTTSEPTEASVMRSYLVGKGIPESAVFLEDESTSTIENLDNSRKIMEREKWDTAVIVTHGFHTYRS
LLMARSLNMNVTAKPVRIKPVKIYYYTLRECAGITYFMGERLLKTIGIG
>lcl|NZ_CP026520.1_prot_WP_042234113.1_917 [gene=gltB] [locus_tag=PC41400_RS04785] [protein=glutamate synthase large subunit] [protein_id=WP_042234113.1] [location=1070033..1074622] [gbkey=CDS]
MNRNASPSKQGLYDPRFEHDACGIGFVANMKGRPSHEIVGQALNVLCNLDHRGGQGSEANTGDGAGILLQIPHGFLSKVC
TEENISLPEAGTYGVGMAFLPQELEARQQCERIAERIIKEEGQQFLGWRTVPTENATLGESAKASEPFVRQLFIARSGDI
QDELNFERKLYVIRKRIENETRALDNTPLYFPSLSSRTIVYKGMLTPEQVDAYYTELQDPSFETAMALVHSRFSTNTFPS
WERAHPYRYVIHNGEINTLRGNVNWMHARQAMIQTDLFGENFQSLLPIVDNDGSDSQILDNTLEFLTLSGRSLPHAAMMM
VPEPWSKHETMDPAKRAFYEYHSTLMEPWDGPAAISFTDGRIIGALLDRNGLRPARYYVTHDDLIVLSSEVGVLSIEPER
IARKERLQPGRMLVVDTVAGRILTDEEIKGQIINEQPYQEWVKKHLVSIGELPEAASEPQPEHDLLVQRQQAFGYTFEYL
RKFLEPMAKTGVDPIGSMGYDAPLAVLSERPQLLYSYFKQLFAQVTNPPIDSNFEEIVTAQETTIGPERNLIRPEAESCR
QIRIPTPVISNEELAKLRYIKREGFKTVTLKTLFRASSGSQGLEQAMEELYTAADEAIADGATFLILSDRGMDGEQAAIP
ALLAVAGLHHHLIRQGTRTKASLIVESGEPREVHHFALLLGYGAGAINPYLAFETLKDMIGNGQIRDITEEKASYNYIKA
STKGVVKVLAKMGISTIQSYRGAQIFEAVGLSQEFVDRYFTWTYTPVEGIGLDGVAQETLQRHQRAYSPRDGRDRVLDAG
GELQWRQDGEDHVFTPETIHSLQTAVRNNDYSMYKKFAELINRESKKAASLRGLLKLKSDRSPVPVSEVEPIEAIFKRFK
TGAMSFGSISKEAHEALAIAMNRIGGRSNTGEGGEHPNRFTPDENGDLRRSSIKQVASGRFGVTSNYLVNADEIQIKMAQ
GAKPGEGGQLPGKKVYPWVAEVRGTTAGVGLISPPPHHDIYSIEDLAELIHDLKNANPQARISVKLVSEVGVGTIAAGVA
KGKADVILISGYDGGTGASPQTSIRHAGLPWELGLAEAHQTLILNNLRSRVVLESDGKMMTGRDVIVAALLGAEEYGFST
APLISLGCIMMRVCHLDTCPVGVATQNPELRKHFMGEPEHVVNFMHFIAQDVRETMAELGFRTLEEMIGRTEFLAQEEEA
GHWKLEGLDLSPLLHQPEMPEEVGRYNQMQQDHGLEKSLDMREILPLCQPALEYQDHVRIELPINNTNRVVGTILGSEVT
RRYGAKGLPHDTIRLHFTGSAGQSFGAFVPKGVTLSLEGDANDYVGKGLSGGKVIIFPPSESTFVPEENIVIGNVGFYGA
TDGEAYIRGIAGERFAVRNSGVRVVVEGVGDHGCEYMTGGRVVILGKTGRNFAAGMSGGVAYVLDEDGTMASRCNTEMVS
LEQVEESYEINELKLMIQYHATFTDSSIAHRVLGRWDEYIWKFVKVIPKDYKRMFDAIERMKRSGLSQQDAVMKAFEENT
RDLARVGGN
>lcl|NZ_CP026520.1_prot_WP_042234111.1_918 [locus_tag=PC41400_RS04790] [protein=glutamate synthase subunit beta] [protein_id=WP_042234111.1] [location=1074666..1076150] [gbkey=CDS]
MGKPTGFIEFQRELPAERAPLQRIQDWKEFYNKNSEEKLQVQGARCMDCGIPYCHTGKIIAGSTTGCPVNNLIPEWNDLV
YRGQWREALDRLHKTNNFPEFTGRVCPAPCEGSCTVGLIDSPVTIKDIEKTIVDRGFEEGWIVPEPPKTRTGKKVAVIGS
GPTGLAAAAQLNKAGHTVTVYERDDRIGGLLTYGIPNMKLEKHLVQRRVDLLAEEGVNFVTNAEIGKNVPVDQLKEEFDA
VILAIGATRPRELDIKGRELKGIYPAMEFLHKNTKSLLDSNLADGEYINAEGKDVIVIGGGDTGTDCVATSVRHKAKSVV
QFQYREKPPETRAVSNPWPEWPKVYKLEYGHEEAKAVHGKDPREYLTLSENFVGDENGNLTGLKTVKVRWRSDDQGRLVP
EEVEGSEKVYPAQLVFLAMGFAGPEQSLLDQLGIERDVRSNAKAQYGKYKTNVEGIFAAGDVRRGQSLVVWAINEGREAA
REVDRFLMGSSNLP
>lcl|NZ_CP026520.1_prot_WP_042234109.1_919 [locus_tag=PC41400_RS04795] [protein=HAD family hydrolase] [protein_id=WP_042234109.1] [location=complement(1076410..1077156)] [gbkey=CDS]
MRQTILFDLDDTLIHCNKYFNVVIQQFVDTMLTWFAGHRLSDEDIKHKQLEIDLAGVEIHGFKAERFPQSFIETYHFYSD
FYDRTPRKEEEEWLYQLGNSVYEHSIEPYPMMDTTLQSLTDQGHELYLYTGGDATIQMRKIQEAGLHHYFGERIFVALHK
TREFMETLIHTQQFERARTWMIGNSVRTDVLPAIEAGINAIHIPAQLEWKYNIVDITATPKGAFFTLPSLTEVPKVISDY
SGKQAVNL
>lcl|NZ_CP026520.1_prot_WP_042234107.1_920 [gene=cls] [locus_tag=PC41400_RS04800] [protein=cardiolipin synthase] [protein_id=WP_042234107.1] [location=1077360..1078790] [gbkey=CDS]
MLWLVIALLLFIFQIVTIVVVEFRNPTKTVAWLMILFILPVIGFIMYYFLAKEYSQRRHVKRKNRRVLGEMQREMRRELE
NTKKGRDYPVPSIVKEPRLFGLLSNIPSSPITRNNEVEVLTNAEAAYEAMFEAMENAKHHIHFEFYIVRHDDTGKKFQEL
LIRKAREGVKVRVIYDGIGSYKLASSYIREFVEAGIDTYCFLPAFIAFFDKRMNYRNHRKIIVVDGTVGFVGGINIGDEY
LGKHPVLGFWRDTHLKLVGDSVYYLQSTFLSDWEFVCRQKLVDTNFFPEHACEGQKPVQIVASGPDAHWDSILESYFGGI
TTAKKRIYITTPYFIPDASLIMGLKTAAISGVDVKIIFPQIPDSWIVHYASLSYIHELMQTGVRFYQYQPGFVHAKVTII
DDLFATVGTANMDMRSFYSNFELNAILYDKESIERLEHDFLEDLKNSKRIILSEFEKRSRFQRGKEVLGRLLSPLF
>lcl|NZ_CP026520.1_prot_WP_042234216.1_921 [locus_tag=PC41400_RS04805] [protein=hypothetical protein] [protein_id=WP_042234216.1] [location=complement(1078869..1079276)] [gbkey=CDS]
MNFKEALKGRTFMLLGLLLLLFILFKMVPLLDAGQPTFWPTIIFVVCILMNMSCDIYLSPFLKKPDERMKAIRSKAYVYS
YFAALIYMAAAAVLVLMEMTPIPVPLIFLDLVAFAGLTPNLLLILFFKRSAKLSS
>lcl|NZ_CP026520.1_prot_WP_042234106.1_922 [gene=ligD] [locus_tag=PC41400_RS04810] [protein=non-homologous end-joining DNA ligase] [protein_id=WP_042234106.1] [location=complement(1079472..1080365)] [gbkey=CDS]
MPTGYKKTQGTIMVEGQELTITNPGKPLWPEAGITKLDFLQKLIELSPYLLRYCRDRHLTTIRFPHGIHDKSFYQKNTPE
PVPDFVNLAPLDGIRYVNLDSLPTLIWLGNLACLEFHPSFHRIGETLPAEWLIDIDPSLEVEPRIMEAAHLIGEVLDSLR
IQSVPKTSGATGIQIYVPIQHGYTFEQLRKIGYFIGSFVVKKYPKLFTIERLKKNRGDLIYFDYLQHWYGKTLSAPYTPR
AKIHGTVSTPLLWEEVERRCDPRDFTLHVIVERIKKMGDLIEKVPPQNLDAVLSHLE
>lcl|NZ_CP026520.1_prot_WP_042234104.1_923 [locus_tag=PC41400_RS04815] [protein=RNA ligase family protein] [protein_id=WP_042234104.1] [location=1080523..1081470] [gbkey=CDS]
MNLPPVVPFEPVSTDRAPSGEEWIAQFKWDGVRVLTYYDGTDVQLFNRKLNERTLQYPEYQAVTDYCQARNVILDGEIIA
LENGKPSFHQVMKRDGLRSGKSLAGSVRKVPVVYMLFDILYCNDEWVIDKPLSERQALLEELVIPDDRVQIVQSFTQTEE
LFGIAQAHGLEGVVYKDLGSTYALKGKDQRWRKHKNYQDVNAVVGGVTYRSGVVNAILLGLYNEEGRLVYIGHAGTGKLS
RQDWIDLTALTRPLIVPEKPFVNEPERSREAAWLQPRLTVKIRFMEWTRYLTLRQPSIQAFLSVAPEECTFLREK
>lcl|NZ_CP026520.1_prot_WP_042234102.1_924 [locus_tag=PC41400_RS04820] [protein=Ku protein] [protein_id=WP_042234102.1] [location=complement(1081564..1082430)] [gbkey=CDS]
MHTVWKGAISFGLVHVPVKMFSATEEKDISMRYIHKRCTTPLSYVRKCTTCDTEVEWGEITKGYEYEPGRFVLFEKEELE
KISGEATKEIKILDFVNLTDIDPIYFQKTYYLSPGDTGANAYNLLLESIKQTGKIGIATISIRSKSSLAAVRVIDNCIAL
ETIFYPDEIRAISQVPNLPEASNVNDKELTMAKMLIEQLSTPFEPEKYQDEYRTALMEAIQQKVAGEEVQVAPEQKRTNV
IDLMSALQASLEAVKPVAADKSDAKAKAKPKAAAKPRKPKETAKETVS
>lcl|NZ_CP026520.1_prot_WP_042234099.1_925 [locus_tag=PC41400_RS04825] [protein=hypothetical protein] [protein_id=WP_042234099.1] [location=1082621..1082818] [gbkey=CDS]
MGDDKQHKELKDGTHQGRDDYYMDIDRMINEGLGGGQVTVHNGLIEESTTDTMGYPESVTDNEQE
>lcl|NZ_CP026520.1_prot_WP_042234096.1_926 [gene=thiL] [locus_tag=PC41400_RS04830] [protein=thiamine-phosphate kinase] [protein_id=WP_042234096.1] [location=1082984..1084015] [gbkey=CDS]
MQHDEFGLIRLLTGDRQQADGMRSQGVVVGIGDDAAVTALSPGMQLVASCDTMTEGIHFLPVTMRDEDIGFKAMASNISD
MAAMGAKARYALVALSLPKDADTEQIRRIYAGLYECADLFGVAVIGGDTTSTRGGIVLTVTILGEVPAGRALLRSGAKPG
DAVFVTGPLGGSAAGLSYLLHRGTDAALWGELPAETAELVRAHTRPLPQAAAGELLQRTGLCRALNDISDGLASEAAEIA
EASGVGLLIAEEKLPLAEGLRACAEKLAADPLEWVLYGGEDYQLLGTAAEADLAALREAFSGEGLPLHVIGRVTAAGEGV
RLQRPDGTVQPLPKRGYNHFAAK
>lcl|NZ_CP026520.1_prot_WP_042234095.1_927 [gene=tsaE] [locus_tag=PC41400_RS04835] [protein=tRNA (adenosine(37)-N6)-threonylcarbamoyltransferase complex ATPase subunit type 1 TsaE] [protein_id=WP_042234095.1] [location=1084031..1084504] [gbkey=CDS]
MAYQYKAADERDTEALAAQLAVLAGPGAVIALDGDLGAGKTRFSQAVAKSIGVQGVVNSPTFTLIKEYEGREFPLYHMDV
YRISLEEAEDLGLDEYFYGEGLTLIEWASLVEELLPPNRLEIYIEHSGGDERVFRLNPSGQPYESWCANLKENGWLV
>lcl|NZ_CP026520.1_prot_WP_241688792.1_928 [gene=tsaB] [locus_tag=PC41400_RS04840] [protein=tRNA (adenosine(37)-N6)-threonylcarbamoyltransferase complex dimerization subunit type 1 TsaB] [protein_id=WP_241688792.1] [location=1084501..1085508] [gbkey=CDS]
MSSGNGNSTREREMAQDTPLAGDSEGRESSRSGFDENGYVLALDTSTASLAVALLQGGRVVAEKHTIVERNHSIHLMPHL
QELLASRGIRPNDLSAVVAGQGPGSYTGVRIAVTAAKTFAWSLGIPAAAVSSLEALALGARTALLAESADAGSDGAVRWF
VPLMNARRGQAFTALYASAGNEPAVELLGSADASGDVAPDNTGSRWARLQPDGIRLVADFARHIGDTAGGTEAAGTTGEQ
SAGDDGTFVVPSGFAKPKEIVFVGETSFFEETIGQTAAELSVPVRVLPYDMTAAEVGLLGLPRLKRGETIDKYTFVPNYT
QLTEAEVNLMAKAKS
>lcl|NZ_CP026520.1_prot_WP_174446851.1_929 [gene=rimI] [locus_tag=PC41400_RS04845] [protein=ribosomal protein S18-alanine N-acetyltransferase] [protein_id=WP_174446851.1] [location=1085579..1086022] [gbkey=CDS]
MRLDDIPYICEIEQEAFTTPWSAGAFHNELTSNHFAHYTVMMVDGVIAGYGGMWLIMDEAHVTNIAISGRFRGRKLGVSL
VHQLQSKARSLGAVRMTLEVRASNYVAQRLYDKMGFQYAGVRKGYYTDNNEDAIIMWADLPDTNQPA
>lcl|NZ_CP026520.1_prot_WP_042234088.1_930 [gene=tsaD] [locus_tag=PC41400_RS04850] [protein=tRNA (adenosine(37)-N6)-threonylcarbamoyltransferase complex transferase subunit TsaD] [protein_id=WP_042234088.1] [location=1086073..1087098] [gbkey=CDS]
MNTTKPVYILAIETSCDETSAAVIENGTVIRSNVVSSQIETHRRFGGVVPEIASRQHVESITWILQEALDQAGLAMKDMS
AIAVTQGPGLVGSLLVGVVAAKALAFSLGLPLIGVHHIAGHIYANQLVHKLEYPLISLVVSGGHTELVYMESEGSFKIIG
QTRDDAAGEAYDKVARALKLPYPGGPHIDRLAQEADGEIPMPRAWLDDSYDFSFSGLKSAVLNVLNQSRMRGEEPVAAHV
AKGFQSSVVDVLVEKAARAVREYGAKQLLLAGGVAANRGLREGLTKRCAELGVPLLLPPMDLCSDNAAMIGAAAFLKWNR
GDITGPEFKAEPVLSLEKWGY
>lcl|NZ_CP026520.1_prot_WP_272509878.1_931 [locus_tag=PC41400_RS04855] [protein=macrolide 2'-phosphotransferase] [protein_id=WP_272509878.1] [location=complement(1087283..1088284)] [gbkey=CDS]
MRFDYLIKNEEVLSDMNDSVKKSTPLTEQEQLNQFRLLAAAYHLELNPSSPMELNDSGLDFLAASARTTDDASWILRIPR
RPDVIQSAAYEAKVLELVGRSLPVAVPGWQINRPDFIAYRKLPGTPAATIDTAAMAYTWHIDPETPSPEFVQSLAAAMAA
LHRIDGRTAEQAGLMAVEARDLRAVAAARMEVVRQAFGVSDDLWERWQAWLADASYWPEHTSLIHGDLHPGHILVDENER
VCGLLDWTEAECGDPATDFTAFYTVFGEKALSELLEEYEKAGGRVWPRMKEHIAQLQGAYAVTIGLFALKSGLEEYAEMA
REALGVPVHKSAD
>lcl|NZ_CP026520.1_prot_WP_042234086.1_932 [locus_tag=PC41400_RS04860] [protein=2-isopropylmalate synthase] [protein_id=WP_042234086.1] [location=1089573..1091123] [gbkey=CDS]
MTKRQIKIFDTTLRDGEQAPGCSLYPEEKVRIARQLVKMGVDTIEPGFPISSPGDFAAVQQISRELQGVEICGFARAVKI
DIDAAVKATQDAALRRIHMFISSSDIHLDFQLRKTREQVLAEARQMVAYAKQFCERIEFTAMDSSRTGREYLYELVEAVI
AEGATIINLPDTVGYAMPEDYGQMFSDVRKHARGADKVEFSTHCHNDLGLAVANSLAAIRNGANYVEVSVNGIGERAGNC
SLEEIVMAIETHKDTLNMETGIVLEEIYNTSTMVSRMMNMPIAFNKPVVGRNAFQHESGIHQDGLLKNRNTYEIMEPEKL
GIPRSMIILGKHSGRHALKHRIGQYGVDLDAQSLEEVYARFKDMADEQKHVTDDELMSLVGDAGASAAERLSLVQLQVVS
SSDRSRVASATIRDNESGAEATWSGTGDGPIEAIVNCIRQAVPMDVTFADLELYSLSASEKATGEATVTVSCGTQTYKSS
GLDVDILFATAKAFLSACNQAVRANPNASQSVSLMA
>lcl|NZ_CP026520.1_prot_WP_042234085.1_933 [locus_tag=PC41400_RS04865] [protein=alpha-glycosidase] [protein_id=WP_042234085.1] [location=1091429..1093180] [gbkey=CDS]
MNREAFYHGTRSNWAYAYDNRTIHLRVRTQRDDVEEIAVIASDKYDWDTFYQEIPMAKTASDSMYDYWEAEVVPSYKRLS
YCFRLSAGSERLWMTETGIHSEQPHPPGGYYAFPYIHDIDLFSPPAWAKDAIFYQIFPERFANGDPGNDPEGVQPWGGKP
ERENFFGGDLQGVLDHLDYLVDLGINAIYFTPVFKSPSNHKYDTLDYTQVDPHFGDNALLKKVVATCRERGIRVMLDAVF
NHSALEFPAFQDVKAKGEESAFADWFHINEYPVAVKNGIPTYDTFGFFEGMPKFNTANPEVKKYLLDVAKYWIEEIGIDG
WRLDVADEIDHRFWREFRQVVKAANPEAYIVGEVWNDSLMWLLGDQFDSVMNYPFSDKVLEFFSGPGMDGHSFASSMGFL
LMRYPKQANEVVFNLLCSHDTPRVLTRVGGDKRRLKLCVVFLMTYMGTPCIFYGDEVGLQGEGDPDCRQCMEWHPDMQDR
ELLDFYKLLIHLRKENRALREGSFRFLKADYGDPCIIYERIDAHTHFTIWMNNTEERVTLQHPMVAEDWRDALSQEEVKP
ADGVMHIELEELGYRILYRTLKK
>lcl|NZ_CP026520.1_prot_WP_042234083.1_934 [locus_tag=PC41400_RS04870] [protein=methyl-accepting chemotaxis protein] [protein_id=WP_042234083.1] [location=1093228..1094979] [gbkey=CDS]
MKYLKRSVTAKMITMMAALVVVLCALIMTLTSLQQRGDYYSRLKSIERVVELGWEQQLPFLTKAMDQMKSTGKPDAAAAE
LEAIRTIMEHYVKDGDMANVALYYPEEKTVDGKMALTVVAANETLDKAGLKPLTDYTFSEQYTQAFQTVKKTGGQGVTPV
YTDQAGEWVTILSPVADEAGKTMAYVGIDFDYGTLSWDLNKRLLMSLAFYVGMAVLFALLLGVFTRRILKPIGELSRLSL
EVAEGNLAVSIPVRSQDEFGTLARNFNAMTASIHSLILGIRESSDQVSHAAEVLDTNAKQTTQATHQIATAIEQVASGAE
RQLQSTQESSNAMEEMATGIQRIAESAGYAAQSSESASDEAARGNEMIQLNVKQMSRMNDTVQRSVESINQLAQLSEEIG
GITALISDIANQTNLLALNAAIEAARAGEHGRGFSVVSDEIRKLAEQSKQSSDKIADLVVTIHTGTGQAVSTMSHTAEEV
TEMTKLVSESGEAFQRIVNHMDELAKQILEVSSSSQQMSAGTEEVAASISELSHMNQQTAESSQTVAASSEEQLASMEDI
SASAGALSDMSRELKQSISRFKL
>lcl|NZ_CP026520.1_prot_WP_042234080.1_935 [gene=abc-f] [locus_tag=PC41400_RS04875] [protein=ABC-F type ribosomal protection protein] [protein_id=WP_042234080.1] [location=complement(1095176..1097131)] [gbkey=CDS]
MLLQVNGIGKSYGTSSVLSNISLQIEDRERIGLVGVNGAGKSTLLQIIAGDLSYDEGDIFKAKETRIGYLRQNGGLQTDK
TIIEEMRTVFTALYAVEKELRELEQQMADPKLVDNAKAYEDTMNRYSARSEWFRERGGYEIEARIRGILHGMGFGEFPPD
TLIQTLSGGQKTRLALAKMLLEEPDLLLLDEPTNHLDIRTLTWLESYLRGYPGAILVVSHDRYFLDSLVTAIYEIERTSS
KRYTGNYSRFVELKEADYEIRMKQFEKQQEEIARMEDFVQKNIVRATTTKRAQSRRKALERMDRIDKPLGDLKRASFSFQ
IDQTSGKEVLQVRDLALSVGEGKRLLDTVSFELRRGETAALIGPNGIGKTTLLKTLIGKLSPDTGSVRWGTNVKIGYYDQ
EQTTLNPQKTILEEVWDTFPTIEEARIRSVLGNFLFSGEDVFKKIASLSGGEKARVALAKLMLLNANMLILDEPTNHLDL
YSKEVLESALLDYEGTLLFISHDRYFLNKMAESMLELERHGVTHYLGNYDDYLEKKAELAEMEAERAREQAARDSKAKAS
SASSAAQQTEKPSSYEADKQAKREERNRQRRIEQLEQDIARLEEEVTALEEELADPAVYNDYVLVQEKNTLIEQKKGSLQ
ACYEEWETLVG
>lcl|NZ_CP026520.1_prot_WP_042234078.1_936 [locus_tag=PC41400_RS04880] [protein=5-formyltetrahydrofolate cyclo-ligase] [protein_id=WP_042234078.1] [location=1097403..1098215] [gbkey=CDS]
MNGSDKKEIRRHIETLRAGLTPEERASRECRVNERLTAWLKGQAARLRLSPDASAGTAPLPLKQPQETEDCPSHVYSGTT
WSAGRETDGSCGSSGLPDLPIVLAYVPFRTECGIHRTIEACWEAGIRVFVPRSLHASRELLFHEIRQWKDLRKGMHGIME
PDPLSPALQDPLAAFAVLVPGLAFDTACGRLGYGGGYYDRFLEKLLQAAREEGRNAPLFAAAAFDVQLVERLPVEPHDIR
LDVLITESAAYTRDPEADRKRGEGSDGAAG
>lcl|NZ_CP026520.1_prot_WP_042234076.1_937 [gene=moaC] [locus_tag=PC41400_RS04885] [protein=cyclic pyranopterin monophosphate synthase MoaC] [protein_id=WP_042234076.1] [location=1098199..1098693] [gbkey=CDS]
MEQPDKLTHFNEQGRAKMVDISEKKETKRTAVARSSVTMLPETLLRIREGRLAKGDVLAVAQVAGIMAAKKTADWIPMCH
PLALSGVDIRFVYESGTELGIEAEVRTTGPTGVEMEALTAVSAAALTVYDMCKALQKDMVIGPTMLVSKTGGKNGDFHRF
DTHS
>lcl|NZ_CP026520.1_prot_WP_009675197.1_938 [gene=mog] [locus_tag=PC41400_RS04890] [protein=molybdopterin adenylyltransferase] [protein_id=WP_009675197.1] [location=1098730..1099230] [gbkey=CDS]
MRWKVAILTASDRGSRGEREDTSAQVIRELVEEEIQGDIIEYRVVPDEMDEIMAALIEMTDYFQADLILTTGGTGLAARD
VTPEATLSVIDRSAPGFAEAMRMKSIEKSPRAMLSRAVTGIRGSTLIINLPGSPKGVHESLMAIIDVLPHALSILTGREQ
EHADES
>lcl|NZ_CP026520.1_prot_WP_042234074.1_939 [locus_tag=PC41400_RS04895] [protein=molybdopterin-binding protein] [protein_id=WP_042234074.1] [location=1099220..1100299] [gbkey=CDS]
MKADPMLREVKVEDAIGLVLAHDLTRIIPGEFKGRLFRKGHQISEEDIPALLSIGKEHIYVLELAEGYLHEDEAALRLAS
AVRGTNTERTEPHEGKVTLKSAIHGLAKIDKAFVDEVNGIGRIVMSTIRTNTVVKPGQALAGTRVIPLVIEEEPVARVER
LAAQRRSGQALTAGSLEPESGGPADAAGTPLPAELVAVKPFRSLRVGLITTGSEVFKGRIEDKFGPVVRAKVEALGSKVI
DQRFTPDDMDAIVTEIRHFIGLGADLILVTGGMSVDPDDRTPGAIARAGARIVSYGTPMLPGSMLMMAYIGSVPVMGLPG
CVMHDPYTSFDVLLPRICAGEEIFREDITEMGYGGLLGC
>lcl|NZ_CP026520.1_prot_WP_042234071.1_940 [locus_tag=PC41400_RS04900] [protein=twin-arginine translocase TatA/TatE family subunit] [protein_id=WP_042234071.1] [location=1100330..1100632] [gbkey=CDS]
MFQNIGFTEILLVAVVCLLLFGPNKLPELGRAFGRTLSEFKKGARDMMSEHDQADKKAAEVKSAAAAPEEVPAPASLTQA
APAPASSERAASDNPRRLPD
>lcl|NZ_CP026520.1_prot_WP_042234070.1_941 [gene=tatC] [locus_tag=PC41400_RS04905] [protein=twin-arginine translocase subunit TatC] [protein_id=WP_042234070.1] [location=1100676..1101431] [gbkey=CDS]
MDKEQPLVEHLAELRKRIFWVLGVLIIGLVIGLVFAKNVILFLKSVPPADKIAWHVFSPWDALRLYMNFGFVVALLITLP
VILYHIWAFVSPGLRESEQRASILYIPGAVVLFLAGLAFGYFVVFPMAFYFTTAISHSMGITELYGAAQYFSFMFNIILP
LSVLFELPIVVMFLTKLRILNPKRLAKMRRYAYMVLVILSTIITPPDAISAILVALPMIVLYEFSVLLSGFIYRNQLVQD
AAWEAEYGEQK
>lcl|NZ_CP026520.1_prot_WP_009675186.1_942 [gene=groES] [locus_tag=PC41400_RS04910] [protein=co-chaperone GroES] [protein_id=WP_009675186.1] [location=1101840..1102121] [gbkey=CDS]
MIKPLGDRVIIEAIAKEETTASGIVLPDTAKEKPQEGKVVAVGSGTLKDGERVELEVKEGDRVIFSKYAGTEVKYEGREL
LIMRESDILAILA
>lcl|NZ_CP026520.1_prot_WP_042234067.1_943 [gene=groL] [locus_tag=PC41400_RS04915] [protein=chaperonin GroEL] [protein_id=WP_042234067.1] [location=1102260..1103888] [gbkey=CDS]
MAKDIMFSEDARRAMLRGVDALANAVKVTLGPKGRNVVLEKKFGSPLITNDGVTIAKEIELEDAFENMGAQLVKEVATKT
NDVAGDGTTTATVLAQAMIREGLKNVTAGANPMVIRKGIDKAVRAAVEELASIAKPIEGKQSIAQVAAISAADDEVGELI
AEAMEKVGKDGVITVEESKGFATELEVVEGMQFDRGYISPYMITDTDKMEAVLENPYVLITDKKITNIQEILPVLEKVVQ
QGKQLLIIAEDVEGEALATLVVNRLRGTFTCVAVKAPGFGDRRKAMLQDIAALTGGQVITEELGLELKSATVQQLGTARQ
IRVTKENTIVVDGAGAKEDIDARVSQIKAQLEETTSEFDKEKLQERLAKLAGGVAVIKVGAATETELKERKLRIEDALNA
TRAAVEEGIVAGGGTALVSVYNAVAAVGSDGDEKTGVNIVLRALEEPVRTIAANAGQEGSVIVERLKKETIGIGYNAASG
EWVNMIEAGIVDPAKVTRSALQNAASVAAMFLTTEAVIADKPEKDKPAMPDMGGMGGMGGMM
>lcl|NZ_CP026520.1_prot_WP_164977108.1_944 [locus_tag=PC41400_RS04920] [protein=acyltransferase] [protein_id=WP_164977108.1] [location=1104299..1105348] [gbkey=CDS]
MFYVHSGFLLFIPFANYYYNANKCVIQPSLKKYFIRRALRILPVYYVFLVLNILLTNNVTILSTEGVKKLFINIFFLQGF
FPEHMINGVTWTLSNEVQFYILLPFIARYFIGRRALIAFPATLALVIIYRIVCFFIYNPNIDSIDWKYYYLTEYNILGCI
DNFTIGMTISSIFYYKKIGKNRHIDNLIISARYLTWLTPLFFGMLFYTYYSWRFKQNSHFSWFFFSFGFDFFLYILFALI
FLYVLFHKSLLTTFLSNTYLRIIGVIGYSVYIWHLPLMDSLNKTNFINGAEGWEKFYRLIPFSLLVILPFSFFMFILVEK
YFIDLSSSLTKKHEDGMASAKPVKNKITF
>lcl|NZ_CP026520.1_prot_WP_042234061.1_945 [locus_tag=PC41400_RS04925] [protein=hypothetical protein] [protein_id=WP_042234061.1] [location=1105686..1106267] [gbkey=CDS]
MVDERHLVYFKELLEGNSRISFRAYLSKNEDSLRKQFSPARFARLKFKSIDEIIKILDEEKISYIVNDQAIRSEKYLATF
HPDALNEKGRLKEEFKDTLFNGIVHDFKTKGEDAILTLYEYIEFPENIHNKKNIEKLEDIEFFAETELCLGDKNLGLFLL
KALASIERQFSDVDDIVLRAKEAAMKHHSSEGN
>lcl|NZ_CP026520.1_prot_WP_241688793.1_946 [locus_tag=PC41400_RS04930] [protein=suppressor of fused domain protein] [protein_id=WP_241688793.1] [location=1106376..1106993] [gbkey=CDS]
MDVQIKRESSKEKNLNFDIRRTIILNLYIKEWSMPEYRVIMTKPETGVYIEIYYFPSSNDKTPARFATVGLSNTIRKTGK
PTETEWMLALEPDLGQESVDRIISYFCDLIAHNIENIKSSETPRVMTESALAPENWTTKALLIDELRGESEDLEKIKIGE
EIISVLWVVPITHKEVHIILQEGIDAFDIYIEKSSYSIIDPTRPY
>lcl|NZ_CP026520.1_prot_WP_042234057.1_947 [locus_tag=PC41400_RS04935] [protein=hypothetical protein] [protein_id=WP_042234057.1] [location=1107081..1107953] [gbkey=CDS]
MTNITEINSLENWQNASRERREDICNSIINKLPGRFQLAKPYLIKNPQNGKEIEIPCFYDELYETPLNLIFGGDFIYGAS
SQHIESIRNIGTPNDWSMISQLIHQSEDNENTHISPFLMSRFPVYEWVLEENINLSNDEERPDFYNEDGPFPTYVTLQEA
KSFLEKTGYRVPWDTEWEYVSKEFNNNLFICGEEIPSRDLSDDICLTQFGDARLNRAASNIFGIAGLAIAAFTRLGGVSD
EIIVRGGAAGFYPFQHPSQWAMLLTELQVPLKNMPGQLAGLRLCLDIPNI
>lcl|NZ_CP026520.1_prot_WP_042234053.1_948 [locus_tag=PC41400_RS04940] [protein=hypothetical protein] [protein_id=WP_042234053.1] [location=complement(1108394..1108855)] [gbkey=CDS]
MLRKICIIFVLVLSTLTFGRSQEESKPLVIPSEYQHAKEMLDHLYNEGLNIQEIHNSKYTAFFNTNPNNSMYIKTDMGIF
ELVHLERKNGKEIDIVVQEATDNGEYKYVVSENGVDRLLILGAENYFNKSDEYITIARNKDLNDKIKQALKAQ
>lcl|NZ_CP026520.1_prot_WP_042234051.1_949 [locus_tag=PC41400_RS04945] [protein=S-layer homology domain-containing protein] [protein_id=WP_042234051.1] [location=complement(1109382..1110722)] [gbkey=CDS]
MKAKIIITFLFIFSLATMPNVGAANTGNLKDISGHWAEAQIKSAVVKGYVNGYQDGTFKPEAPVSGEEFLAMMVRALKIP
TESQLSEESWFSPIYKSAVKSGIYQGDYQKGWDQPLTRGDMALTLERATRQRSFKQLVESSPQSRIQDIDSMHKNMEPKQ
MVYESVRRGLMTGTSSTELSLDSATTRAQAVVVIERVISYNAGESLPIDRHTLEVAEVFWHSTNLFTVWSNFLGIENANK
FDTTKLHMETDFFVGDITGLYIIDANDPDSPYWDKIQYPLSELYRNSYKEAYPVEDIKDAYYIVHSISENEIENSQHLAP
ININIHGVGEDPVTYFNAGDVSAMRERLENGSFEVKEVLLLRKHTVPKVGEAQHHINGKYFDSKVAVRTYIVPKYNVLVD
KITIDVSSTMIGDRKNQRVLNIWPTGIRTLKPNASDVELYMLEELN
>lcl|NZ_CP026520.1_prot_WP_042234049.1_950 [locus_tag=PC41400_RS04950] [protein=MarR family transcriptional regulator] [protein_id=WP_042234049.1] [location=complement(1111355..1111753)] [gbkey=CDS]
MKFITSNKHFLKQADESIKELGLNSQQGKLIAFIHEHQEQGLIQKDLSERFHVRGASITSMLQGLEKKGYIERKIPAHNE
RQKNIYVLPKGLALINDFKLSFQRVEEKIVQNLNEDDQQVLQQLLTKINQNL
>lcl|NZ_CP026520.1_prot_WP_042234047.1_951 [locus_tag=PC41400_RS04955] [protein=Crp/Fnr family transcriptional regulator] [protein_id=WP_042234047.1] [location=complement(1111815..1112513)] [gbkey=CDS]
MTKTESANNSSVTSWLEGLAFQWHPLPQLGQKLVIQKHAFIFFENQPNDFVYLVLNGRIRLYVTSLNGVEKTIAIIGKNG
MIGENGIFDNAASYNHSAIAASKATLIKIDRKKLEETLKSNFELTRQVLLWVNLKMRIMSSQLLQLSSYSATQRICHTLL
QLVDTYGLKNGEKEFIGIGFTHQELANLVSTSRVTAANTLKNLELTGVISKQNGKYVIEDKARLISLGTHHN
>lcl|NZ_CP026520.1_prot_WP_042234045.1_952 [locus_tag=PC41400_RS04960] [protein=aminotransferase class V-fold PLP-dependent enzyme] [protein_id=WP_042234045.1] [location=1112555..1113688] [gbkey=CDS]
MEFPFVKERQYFPILKNKIQLSSCSQSAMSLQVEAAMKEYTDGWLTEGMDWTGWVHRVEMAKQSFATLINASPDEIAVLP
SVSLAASAVASALQFTTGRNKVVTTEMDFPSIGHVWLAQQSRGAQVAYVPSENNEIPIEFYDRYVDEQTLIVSMSHAAYY
NGLKQDLKQISAIAHGKGARLFVDAYQSAGCAAIDVKEMDVDFLAAGAQKYLLGIPGIAFLYVKKELADELQPSITGWFG
RIDPFSFDVKSLDYAEGTRRFDTGTPPMINAYAAYAALELINQIGVHRIEPYLNELSNFAVSYGQEKGLTIASPLDLKRK
AANTAFHCPNASLIEMKMRERGVIVSARNDVIRIAPHFYNKKEEVALAIDMLYELKS
>lcl|NZ_CP026520.1_prot_WP_042234043.1_953 [locus_tag=PC41400_RS04965] [protein=amidohydrolase family protein] [protein_id=WP_042234043.1] [location=1113754..1114995] [gbkey=CDS]
MANKKIFKNALLIDGKSNTVRKEALVMVEDERITYVGSYDETIVQQAGEAELFDLKGMTLMPGMIDTHVHLCMAGEASTF
TDMVMDTPAFAAIKGVERAKRSLEAGFTTLRTMGEKGHIDIAIKKAVEQGIIVGPTIFASGKALTITGGHGDMFPEGVNV
DGIGIVLDGVDQARRAAREQLKLGADNIKMMATGGGMSPGPGTIAQLTIEEMQAAVEEAVKFEKITAAHAIGVEGIHNAL
KAGVRTIEHGTFLDDQGIELFLEKNAYLVPTLSAFKTIKYGRAGGVPEHHIKKVEYYQTAHEANLQKAIAAGVKIVAGTD
AGTPFNYHGENAYELECLVRNGLSEMDAIKSATQVAAEALRLTELGVIEAGKIADMIVVDGNPLHDIKILQDKARIKRVY
KGGQLVHINSAEC
>lcl|NZ_CP026520.1_prot_WP_042234042.1_954 [locus_tag=PC41400_RS04970] [protein=DUF3311 domain-containing protein] [protein_id=WP_042234042.1] [location=1115033..1115245] [gbkey=CDS]
MRNYYWLVIIPFIGMLGGAVVFNQVTPYVLGLPFLLFWILLWIVLSSLTMLVIYQLDKSNGLLDKEGDEV
>lcl|NZ_CP026520.1_prot_WP_042234041.1_955 [locus_tag=PC41400_RS04975] [protein=sodium:solute symporter family protein] [protein_id=WP_042234041.1] [location=1115242..1116723] [gbkey=CDS]
MNTALLIMLGFLLLSLFLGLMARRGKDMNMEQWAVGGRGFGTAFVFLLLAGEIYTTSTFLGTSGWSYSKGGPAFYILCTS
SLSYLLGYWLLPPIWRYAKEHKLVSQSDFFAIKYKSKGLGVLVALVGVLALIPYLILQFKGLGIIVSAASYGSISPSLAI
VISTLVITVYVSVSGIHGSAWTAVMKDILIFIIVLFLGIYLPYHYYGGIQSMFEAVNEAKPNFLIVPEKGLSFSWFISTT
LLTVIGFSMWPHTFAAYFSAKSTNVFKKNTIIMPLYTLMLPFVLIVGFTAILQVPGLTGPNGDLALLKISIQTFDPWFVG
FIGAAGLLTGVVPGSLLLMAASTAIAKNIYQVIFPATTDAQLSRVAKLIVPVIALVCMFYAIFNSSTIMTLLLMGYNFVT
QLFPPLLASLVNKRFFTKQGAAAGMLVGVSTVIIITLNDLTIAQLFPGLPQAIQELNIGFIAVLLNTAVLILVSMLTRKQ
AIHTNNVGLEKGA
>lcl|NZ_CP026520.1_prot_WP_042234039.1_956 [locus_tag=PC41400_RS04980] [protein=amidohydrolase family protein] [protein_id=WP_042234039.1] [location=1116726..1117919] [gbkey=CDS]
MKTYIQNVTLYNGRGSKTEGANILFDETGILGLGTHDLTSQADLIIDGTGLTALPGLIDLHVHLTMDGSPDSMGKTIADS
AGVAAYRALVSAVKQIRSGVVTVRNCGSKYNIDIELRKAIAEGIVKGPRIFACGQPITMTGGHCHMFGTEADGVEEVRKA
ARSKIKEGADFLKLMATGGGLTPGVKAGASQLTEEELAVACQVAKEAGKRTAAHAQGNEGIKNAIRAGVTTIEHGVELDD
EAIQLMKEHNTYLVPTLAAPYQIVKNGTAAGIPEFAVRKNEEAMIPHRESFRKAHMENIKIAAGTDAGTPFNLHGDFATE
LELMKEEGMSLQEVIHAATYMAAEALGIEGETGSLEIGKWADVILLEGDPEADFSAFRRVTHVFKSGASMYMHALEV
>lcl|NZ_CP026520.1_prot_WP_255246866.1_957 [locus_tag=PC41400_RS30540] [protein=hypothetical protein] [protein_id=WP_255246866.1] [location=1118301..1118432] [gbkey=CDS]
MTEIIKAIVPLISNLLLAVGHFFLHKKIDKTNWKVPQREETTN
>lcl|NZ_CP026520.1_prot_PC41400_RS04985_958 [locus_tag=PC41400_RS04985] [protein=DUF4352 domain-containing protein] [pseudo=true] [partial=5'] [location=<1118490..1118627] [gbkey=CDS]
FFLNKLNPGLSATGYVVFNVPADAKGFVLEASGGFTGKKIKLKVE
>lcl|NZ_CP026520.1_prot_WP_042234212.1_959 [locus_tag=PC41400_RS04990] [protein=S-layer homology domain-containing protein] [protein_id=WP_042234212.1] [location=complement(1118704..1120335)] [gbkey=CDS]
MNRWFTKTAQMLCLTLMLSALMPILASAQTSYSGVSFNNGTVKGSVYLDHKNGNETELLLYGPSGQYVGVVKATYADYFD
EAKQQYKYEFQTVVDSAYRYVKLYSVVDSVYATVDNEAYNSGNDNDSDNGSTTPGGSTGGGGGNDGGVTTPGDGSTAVNS
IKAGSDGSLDAQALKDALAKGDTLEIVLNGDMVNIPAWVLKENSDNKQITVRNGSAVYYVPLATINYDSLASSLSVGASD
LYIRVTVKFVSESMRSTMQTEAAAMGGKIIGTPADFNLEAATTAGKTSMINTGNMYVPRSLNFDTVLNQAKTTAFLYNET
TKTMTFVPAFYTFSDGKTDVMVKRQGNSIYGVLELNKTFSDIKGHWAESNIMMLANKLVVDGVTNTTFQPDRSVTRAEFA
AMLVRSLGIQNTGGSSTFKDVASNQWYADVVATAAKLKLVDGYEDGTFRPNNQITREELSAMVVRALSYAGVNTSTEITK
GQAMLSKFKDSHTIIWAKDEIAAALSLGIIDGMTSTTIEPLQQATRAQSATMLKRFLTAAKFI
>lcl|NZ_CP026520.1_prot_WP_042234037.1_960 [locus_tag=PC41400_RS04995] [protein=O-antigen ligase family protein] [protein_id=WP_042234037.1] [location=complement(1120657..1123110)] [gbkey=CDS]
MYAYKKPKLDEREQAEKSSIIFWLLIGFVTLFLFWAPFQKALFNGNSLNFERPIYSAVIWASIFLFLTGIYFFYFWKLRE
RKDLLSLFIFLIPLSYLISLLPAASKYSAVNLLLIQIVLVTLFLIGAYLTKSALGNKIIHIGLLTSGYVIVIFGLMNWLG
NGKLAGVIVGWFSTIDATGIYKDAVMTDSNGLRLTSVFQYANTYAGFLIALWGTGLFLVSKNRSWSKVLVHALFLVPVLV
SFMLTLSRGAIIFLPIIFFLILIFQKPTRQILLLVHAILSTLASFAVLSKITAIGIDVNKEFSASLSGKGWMLLIGASVI
YGLLALLVQRFIAPPLESKLGNFSAKKWALFAIPAGSIVLGILGVLVVMATPLKNLLPENVRIRLETINFTQHSVLERIT
FYKDASKLFADYPLFGAGGGAWSSLYEQYQNNPYTVRQAHNFFMQYLVEVGAFGILILAVLLVAIFAIYIRNYIRSTEQE
RDKHFVYFIFVIAILGHSVLDFDMSYVYIGALVFLCLGAMIADVKTKEFTKQVVSIRYVYPSVILVLSIITLFVSARYLS
ANSSYNKALEVAKTSQNYTEIVTPLNNALSLRANHPEYVLLKVDLLAQLYKQTQDEKYLNEEIALINTARVREPYNKSLL
DRELNTLQIKGQQDQVEKLIQNGLNYYPWDISFYERSIQTHVTAGFEGYAAKDKTTADKEWNQAILVYQQVLDRMKELAK
LPEGQLQGREFDVTEPMRLYLGQMYFAQGKYADTEALLKPFSAGELKDENGQTGIRYYLAALKKQNKTDQPALDKVKALT
PTVEQEVDQIIASTPPL
>lcl|NZ_CP026520.1_prot_WP_241688794.1_961 [locus_tag=PC41400_RS05000] [protein=ABC transporter permease] [protein_id=WP_241688794.1] [location=1123239..1124093] [gbkey=CDS]
MSKQGVFKSSSLLRNLKLQIKKPASLLKLNKLLIYQLIKREIAVRYRGSKLGMLWSFLTPILMLVIYTFVFSIVFKARWG
TSNDNNIEFALILFSGISVFNLFSEIIIQSPRLIIGNVNYVKKVLFPLEILPLTLIGSALFHYVIQLLILLLGILFTYST
IHWTIIFLPLVLLPLLLVALGLSWFLASLGVFIRDIGHIITLFMQGLMFLSPIFYPVSSVPEKFRALLFLNPMSYVVEDI
RKILIWGQMPDWGFMGTSLLLSTVTVVLGYIWFVKTKGGFADVL
>lcl|NZ_CP026520.1_prot_WP_042234035.1_962 [locus_tag=PC41400_RS05005] [protein=ABC transporter ATP-binding protein] [protein_id=WP_042234035.1] [location=1124083..1125291] [gbkey=CDS]
MSYDLAVKITDVSKLYKLYDRPEDRLKQFFSSNKNKFNKEFWALKNVDLTVNRGETIGLIGKNGSGKSTLLQIIVGTLQP
SSGSVDVNGRVSALLELGSGFNPDFTGRENIIMNSAILGLNAEEIKEKLPDIISFSGIEKFIDQPVKSYSSGMFVRLAFA
CAINVNPDILIIDEALAVGDIQFQLKCIEKMKEFKKQGKTIIFVSHDLYSIKNFCDQAIWMMDGQIHMRGDVHSVSNSYE
NFMKTSPSQHTEEEEIEEDQQDDVLKINGMEFYDTQFSDKKRFNYGEKLSIVVNYTLKKKVEGIVGGIALYDSQDNYICG
LNTKLDKFSLPSEPGDYQLALHYEQMNLLPGSYFVDVGFFESSGIVRLDYKKKMDFFTISTSEYFAEGITCIPHNWNSRK
EA
>lcl|NZ_CP026520.1_prot_WP_042234033.1_963 [locus_tag=PC41400_RS05010] [protein=methyltransferase domain-containing protein] [protein_id=WP_042234033.1] [location=1125293..1128064] [gbkey=CDS]
MKYDFKINTSIQNAHTIVLQHIRPETTVLEFGTATGYMTEYMYKELNCQVYGVEIEESAARVASAFTVDMEIGDVESLTW
TTKFKNIKFDHIIFVDVLEHLKDPWRVLKIAVEFLKEEGTLITSIPNISNNAVIMELLSGRFEYKTLGLLDNTHLRFFTK
SSILELYEQAGLQPIGWYATYMVPEQTEFKQSYENYPLVRDFLQQRTDGEIYQYITVGKKTENAVKPNQAVIEQRYRLSS
YYLQLFWSENEEFTEYDSVHLPLHIDGSTITYDIPLPNKEINKLRFDPISELALIKLSEIYLLETVCDTENGIPSYIQKW
SNENNFEGCHFYNLVDFGGREPLISLNKDPYIILNPTTKIKKDKTYFLRIKMSVSVKNSDLVEELNRQYSKIKREAERLG
LLNMSLEEEQRKQNKLLFELEHIKELLQEKYTKDIETFKSKILQLEAQLEVKNVALEEHRKNISTQQVYYESEIQKREEA
TEKVLNELQVLLNSTSWKYTRFFRAAMNSLRELKNSAKHRVKNMIVKHKKKSNKKEMINLLDKERYILVFSHTNYLNSMG
GTEKYIYEQASYNARSGIGTVQIFPSNRYDFYAEEESFYGINVGDTFLGYLTAQEIVEFIQDSAHKYKKTYIHHLLFWIY
SDFKNIFEAVNYLGISSVFVAHDFFACCSSYHMMYSDSNGKKGCIPDLMTHGTENVCIKCDHYADLEKRKEIFNDIFNVC
EKIVLPSQYVLEVFKLIYPKWEDKFLVHGHLRLVKDGTSEKEKVNEKIKIAYLGYKMENKGWSTWLKMINNEQFSDLYEF
HHIGSSEHYSTNVRSHEYSFINDGRMAATNLLKKENIDIVLLWSIVPESYSYTLYESIAAGVPVVTNKHSGNIAFSVQNS
SEDIGVVLNNEKELWALLSDNNSVLNLVNRARNLYELEYNELD
>lcl|NZ_CP026520.1_prot_WP_042234031.1_964 [locus_tag=PC41400_RS05015] [protein=hypothetical protein] [protein_id=WP_042234031.1] [location=1128080..1129153] [gbkey=CDS]
MIVTTSICANYLPKAMVLAKSVKEADPSIKVVVSLLEREVTPEAQAFEYFDDIVLGKDLGFENFEKFIFKHSIVEASTAV
KGQLFLYLMENYQEETKFVYLDPDIQVLSSLTELEEALESNPIVLTPHLTTPEDIMDAVMDNELSALKHGAFNLGFLAVS
RSEESKRFIEWWASRLDMFCYDDIPRGIFTDQKWMDLAPCFFDVYILKHPGYNAAPWNVSKRKIHINSNNEYIVNDKFLR
FFHFSGFDSGANEGMINKYVPDKKNAIYKLRDEYVKEMNLMGQEELGAIPWSYDYFNSGEKIERTTRIAYRDNLEMPYKF
QDPFNNSNAELNATSEQIEKKENFVKGIKKVVKKFIN
>lcl|NZ_CP026520.1_prot_WP_042234029.1_965 [locus_tag=PC41400_RS05020] [protein=sugar phosphate nucleotidyltransferase] [protein_id=WP_042234029.1] [location=1129251..1129994] [gbkey=CDS]
MKGIILAGGTGSRLFPLTKVTNKHLLPVGKYPMIFYSVGKLKQANITDILVVTGKDHMGDVVNLLGSGHEMGVTFTYKVQ
DEAGGIAQALGLAEHFASGDQVVVVLGDNVFEDNISTYVQNFREQGAGAKILIQEVHDPQRYGVAELRGNHIVSIEEKPL
NPRSNFAVTGIYMYDSLVFDIIKTLKPSIRGELEITDVNNAYIERGQLTYDVLNGWWTDAGTHGSLAKANELAKDIIFGE
EFGKLKL
>lcl|NZ_CP026520.1_prot_WP_042234028.1_966 [gene=rfbC] [locus_tag=PC41400_RS05025] [protein=dTDP-4-dehydrorhamnose 3,5-epimerase] [protein_id=WP_042234028.1] [location=1130012..1130581] [gbkey=CDS]
MNIIPTKFSQALLLEPTVHGDNRGFFMESYSQKRFEDLGITYNFIQDNHSLSSETGVLRGLHYQLNPKAQTKLVRVLVGG
IYDVIVDIRKGSPTFGQWQGFILSANNKRQLLVPRGFAHGFCTLVPNTEVLYKVDEYYSPENDRGILWNDPALGIDWPTS
TPVLSDKDRRHPLLQDAELNFEYEEELHP
>lcl|NZ_CP026520.1_prot_WP_042234025.1_967 [gene=rfbB] [locus_tag=PC41400_RS05030] [protein=dTDP-glucose 4,6-dehydratase] [protein_id=WP_042234025.1] [location=1130578..1131582] [gbkey=CDS]
MKLLVTGGAGFIGSNFVLYMLRQHPSYTIINLDALTYAGNLENLKSVENHPNYRFVKANITDQRAIQDIFRQGIDVVVNF
AAESHVDRSILDPEIFVRTNVMGTQVLLDAAKAYGVQKFIQVSTDEVYGSLEETGLFTEETPLSPNSPYSASKAGGDMLV
RAYHETFGLPVNITRCSNNYGPYQFPEKLIPLIIANALDDKSLPVYGDGQNIRDWLYVEDHCSAIDLVIHHGRNGEVYNI
GGNNERTNLAIIKTILQELGKPENLITFVPDRPGHDRRYGIDPSKIMLELGWKPKYHFESGIKETINWYLQNRSWWENIR
SGAYQDYYTRQYKE
>lcl|NZ_CP026520.1_prot_WP_042234023.1_968 [gene=rfbD] [locus_tag=PC41400_RS05035] [protein=dTDP-4-dehydrorhamnose reductase] [protein_id=WP_042234023.1] [location=1131602..1132456] [gbkey=CDS]
MKVLVTGANGQVGSEVCRIFETGSKKIYGYGREQLDFTNQKVCQRVIQDIVPDAIIHCGAYTAVDMAESEADLAFKINAM
GTRNLAVEAEKVKAKFCYISTDYVFDGKSDVPYTEYDNTNPQTVYGKSKRAGEIMTQTLCSRYFIVRTSWVYGRTGKNFV
KTMLNLGREQSVLKVVDDQFGSPTSASDLARFLQELIETECYGIYHASNSGICTWHQFAEAIFKETGMTDVKVLPCKTDE
FLRPAQRPQYSVLDSMAIRINNLGELGPWEKSLARFLKEFKNES
>lcl|NZ_CP026520.1_prot_WP_042234021.1_969 [locus_tag=PC41400_RS05040] [protein=glycosyltransferase family 2 protein] [protein_id=WP_042234021.1] [location=1132446..1133381] [gbkey=CDS]
MRANDYIAVLLSTYNGQDYIKEFLNSLYNQTVDDFMLYIRDDGSKDDTRKIIDEFKKDKKNIIFLEDNNNLGPVLSFNSL
LEVALENKNIKYFMFADQDDIWLIDKIETSLSTIKKEEENSGQLPLLVHTDLTVVDQKLEVIGLSFFEYQKLNPKRIMFN
HLLMQNVITGCTVIINRELAKVCFPISSSAIMHDWWIGLTASVFGKVVYLNRTTILYRQHSGNSIGAKKYDIRYLFERLT
HKVTLIKNIYQARSFHNNYADYMDESKRYILEQFLSLKDSTFLIIRFKLIKNKFLKHGILRNVGLLLRLKN
>lcl|NZ_CP026520.1_prot_WP_042234018.1_970 [locus_tag=PC41400_RS05045] [protein=DUF6056 family protein] [protein_id=WP_042234018.1] [location=1133393..1135135] [gbkey=CDS]
MTYKNITKYILYSILLLFFFYFLIINILMPLTPDDFSRATVSHTGKKITNIWDIVYSQYYFYQVVNGRAIVHAFVELFMM
LGKDLFNIINSLVFISLCVLILIHSGIKEINKFVGFHFILVVILLWFGLPSIGDTIFWLSGSVNYLWTAVFSLSFLLPFR
FAIEGKELLNSNKKAHILMLVLGLLAGWSHENTAMLTGCFLFFIFLYFWFGKIKIARYLYTGALSFVIGALFLFLAPANF
NRVKSMGYTDNYDLWRQVKIGFESIKYILKSSQFIFVLFFFLILIYIWTSKRSRVKNNKAIIQVLIYFSAACVSIIPMLV
SPEFPPRSFFASATFVVISIVMLYRVISTEINSSLKVISLFILGILLVMSMSRVYLDYNKIHETHTAREALITESKQKGI
YSVSLPKMPLATDKYIFVYEINKSPDYIANGVYAQYYDLDSVVLEGDYLKVYTTDDSLAGWYKIYYDSGQSFREKESTYV
IHATALDKNQLYFNYPETEIKQLRLDFPASATANVTIEKIVINQNGVAKEIDMDKLGKNIQPVSGILNYSVDKSVLKISI
EGENPHLIIRNFNELVETTK
>lcl|NZ_CP026520.1_prot_WP_042234017.1_971 [locus_tag=PC41400_RS05050] [protein=glycosyltransferase family 2 protein] [protein_id=WP_042234017.1] [location=1135155..1136111] [gbkey=CDS]
MKFISFIVPCYNEEQTINIFYKQILEVMQRIDVLYEIVFINDGSKDSTAEIIRRLCKEDPSISFISLSRNFGKESAMFAG
LEHAKGDAVIIIDADLQHPPALIPQLVDGWQNEKYDVVYARRINREGEPILRSWFSKKFYSIINKISDVKLVDGVVDYRI
MDRKVVDSILGLRERHRFSKGLFEWAGFHSKCIEYKNVERAAGNSSWSFNKLFHYAIEGIVSFTIVPLRIATFVGILTAL
AAFIYMGITIIQTLLFGNSVSGYTSTISLILFFGGMHCIFLGIIGEYLGRIFNEVKHRPHYFIREHIESRVRTFEFKK
>lcl|NZ_CP026520.1_prot_WP_042234015.1_972 [locus_tag=PC41400_RS05055] [protein=GtrA family protein] [protein_id=WP_042234015.1] [location=1136095..1136496] [gbkey=CDS]
MNLKNRFFSAEFIRFGIVGVFNTLHHYLWFFVFGVFLNYDFANVFAFIISMIGSFYLNCYFTYKVRPTFKKFIKFPLVYL
VQIIMAYLIPKIGVEIFNVKEIFVPLLTSFSIPVVFLLSRYILKKEEKKIMKL
>lcl|NZ_CP026520.1_prot_WP_042234013.1_973 [locus_tag=PC41400_RS05060] [protein=glycosyltransferase family 2 protein] [protein_id=WP_042234013.1] [location=1136515..1137414] [gbkey=CDS]
MDLSIIIVSFNTRQITLNCLQSIYSSIVGYTFEVFLVDNSSTDGTTTAVMNTFPKVRIIKNKTNVGFSRANNQGIKEADG
RYILLLNSDTILKRDTLQIMLDFMQASPEVGASGCKVVLPNGTLDKACKRGFPTPSASFYYAFGIARLFPDIPRFNGYQL
SHLNPDEEYPVDCLVGAFMLVRRETIDEVGLLDEEFFMYGEDVDWCYRIKQAGWQIYYYPRTEITHYKGASSRRKPFKII
YEFHRAMFLFHRKHYKRKYSFLTNVAVYIGIGLKFTLAFLKNKALPALAAPEKKQDTGS
>lcl|NZ_CP026520.1_prot_WP_042234011.1_974 [locus_tag=PC41400_RS05065] [protein=undecaprenyl-phosphate glucose phosphotransferase] [protein_id=WP_042234011.1] [location=1137488..1138903] [gbkey=CDS]
MLRRNQQFLTQVYAIFDFAVIQTAFLIAWWVKFESGWLDANQPIPISVYAAWSAVYGAVAVAVGFFVGLYGPNRKKRFAD
ALIKIVQTHLISLFIQLSLLFFFKEVHISREFLAMYIVLDIVLITVYRYVLKESLKRLREKGYNKQFVLILGAGTLGRRF
HDNLLQHPELGYEIIGFLDDHGGHSDHKELPAAPYKPVIGSLDDLDEILQDKLIDEVIIALPLDVHHKFPRIIAACEKAG
VRTLIIPDFFDYLPARPFFDNFAGMPLINVRDIPLDMISNRVIKRAFDLAFSLTAILITLPVMIVIAVAIKLTSPGPVIF
KQERVGLNRRKFYMYKFRSMRQETANTSDTGWTTADDPRKTKLGAFLRKTSLDELPQFFNVLFGHMSVVGPRPERPFFVE
QFKEEIPKYMVKHHIRPGITGWAQSNGLRGDTSISDRIQHDIFYIENWSFLFDIKIIWKTIVNGFTNKNAY
>lcl|NZ_CP026520.1_prot_WP_255246865.1_975 [locus_tag=PC41400_RS30545] [protein=hypothetical protein] [protein_id=WP_255246865.1] [location=1138966..1139094] [gbkey=CDS]
MIDHEDAYRDDQMPMGRGLIIGVLMGAVMWAVIIGGFLYLMN
>lcl|NZ_CP026520.1_prot_WP_129112454.1_976 [locus_tag=PC41400_RS05070] [protein=outer membrane lipoprotein-sorting protein] [protein_id=WP_129112454.1] [location=1139474..1140895] [gbkey=CDS]
MRRVTWILAIVMSAALLLAGCGKKDAGSVVKDLDHVIGKLESYQGTGRMVLQTGQEPQEYQVEVWYQQPHYYRISLTNEK
KDITQVVLRNDDGVFVLTPHLHKSFRFQSDWPENQGQVYLFQSLAQSILLDTERQFTKEGNSFVFDVAANYANSGQMARQ
KIWLDQKSLAPQKVEVSDSSSNVLVKLDFTQFEFGKKLEKDSFDMQRNMTSWDLKSLDKDAPVSGAATEPSAGAASPDGK
APAAGKDTPASAQAGAAGKDAAAGTPASGASAGAAADGGKAPAASPGKAAAADGSAKAPAAATGSGGAAADAGKAPAASG
ASAGGAASAGGTGDKQPAPAAAGTAATGAAGADNKAAANSSAGFGLIEPSYMPKGVAQQDIRDVKLGENTGIMVRYQGTY
NYTLLEVAPQTQTVSVLPGDIVDLGYTLGVVTGDEKKTLTWTYEGIEFRLATADLPHDELVKIAQSVAEEAGK
>lcl|NZ_CP026520.1_prot_WP_042235933.1_977 [gene=alr] [locus_tag=PC41400_RS05075] [protein=alanine racemase] [protein_id=WP_042235933.1] [location=1141078..1142277] [gbkey=CDS]
MDAFYRPTYVEISLDALRSNLETFRRVLPGSVHMMAVLKADAYGHGAVEVSKVAMECGATYLAVAFLDEAMELRNAGITA
PILVLGYTPPEGIGLALEHDITLTVYTEELLEAASGYAQRDAAAGGNKPPLNIHIKIDSGMGRLGLHDEDEAVAFIAKAL
QTRGIRVEGLFTHYANADETDKAYTYEQHAKFAGIVRKAGELPGARFPFVHAGNSATAIDLPELTYNMVRLGIVMYGLYP
SEEVNKERIRLKPVLSLKTGIVHLKTLPSGSGVSYGTIYHTQGDERIATLPIGYADGFSRLLTGKAEGLVRGRRAPVVGR
ICMDQCMLSTSDIPGAAFGDEVVLLGEQGGERITAEEIAAKLGTINYEVTCMISHRVPRVYVSGGRRVKAVNSLFRHLF
>lcl|NZ_CP026520.1_prot_WP_036740450.1_978 [locus_tag=PC41400_RS05080] [protein=CopG family ribbon-helix-helix protein] [protein_id=WP_036740450.1] [location=1142493..1142774] [gbkey=CDS]
MNNAHNTKRIMISLPDNLLQEVDGIVEKENSNRSEFIRQAMKLYLLERKKRSLRESMQKGYMEMAKINLTIADEAFQAEE
EADNTLVRLVSGV
>lcl|NZ_CP026520.1_prot_WP_009675190.1_979 [locus_tag=PC41400_RS05085] [protein=type II toxin-antitoxin system PemK/MazF family toxin] [protein_id=WP_009675190.1] [location=1142779..1143129] [gbkey=CDS]
MIVKRGDVYFADLSPVVGSEQGGVRPVLIIQNDIGNRFSPTVIVAAITAQIQKAKLPTHVEIDAETHGFDRDSVILLEQI
RTIDKQRLTDKITHLDEESMRKVDEALQVSVGLIDF
>lcl|NZ_CP026520.1_prot_WP_042235930.1_980 [locus_tag=PC41400_RS05090] [protein=Tex family protein] [protein_id=WP_042235930.1] [location=1143312..1145543] [gbkey=CDS]
MANTDVQHADKQQAPKSYERTEEKTAATAARIVKQIAAELQLKASQVQTTVNLLDEGNTIPFIARYRKEMTGELDENFLR
DIEERLNYLRNLEDRKLEVLRLIDEQDKLTGELQASIEKAVKLQELEDLYRPYRQKRKTRASVAKEKGLEPLAKFILKQP
QKGSLQEEAAKYVREDKGVATAEEALQGAQDIVAEQLADDAKIRAWVRKFTHDNGILRTEAKKQEEESVYEMYYSYQEPL
KKMPPHRTLAINRGEREDVLKVALDVPVERIHEFIGRQVIQGPSVTRDALQSAVEDAYKRLIAPSIEREVRGELTERAET
HAIQIFAGNLRNLLLQAPVKGQTVLGVDPAYRTGCKLAVVDETGKMLEVTVTYPTPPNNKVAEAKAAFKRLIDTYDVSLI
VIGNGTASRETEQFVADLIAELKARKLQYLIVNEAGASVYSASKLAQTEFPDLDVAERSAISIARRLQDPLAELVKIEPK
AIGVGQYQHDVSQKHLDESLKAVVESAVNHVGVDVNTASPSLLSYVSGINSTIAKNIVKFREENGKFGSRKELQKVPRLG
AKTFEQCIGFLRVPGSGNPLDNTPIHPETYDVVDKLLAELRVELAGIGSQQLKELLQRQEPEELAPKLGIGVPTLRDILD
SLQRPGRDPREELGTPIFRTDVLQIEDLKPGMELQGTVRNVIDFGAFVDIGIKSDGLVHISQLSDKFVKHPMDVVSVGDN
VTVWVLNVDTKKGRVGLTMKQPQ
>lcl|NZ_CP026520.1_prot_WP_042235927.1_981 [gene=nagZ] [locus_tag=PC41400_RS05095] [protein=beta-N-acetylhexosaminidase] [protein_id=WP_042235927.1] [location=1145738..1146997] [gbkey=CDS]
MLHTWLRKGGTLALAVLLFISAGCSTSGPQAGPGKSQSGGKESQPSPTPTVDDPLKDQIARMTLDQKIGQMVISGVDGLE
MDEHARQLIDTYKVGGFILFKPNVKSAAQAVALLNSLKEENALTGSKIPLMLSVDEEGGRVSRMPDEYMNVPASRVIGST
KNLRFAAEIGDILAREVTSLGFNTNFAPVLDINSNPNNPVIGDRSFGSAPELVGNLGVEMMKAIAAQKVIPVVKHFPGHG
DTSVDSHIGLPVVEHDANRLRSFELVPFAKAFQSGADAVMVAHILLPKLDSTYPASLSSAVITDLLRKELNFGGVVFTDD
LTMGAIVKSYGIGEAAVQAVKAGTDVALVCHEFDNTVRVVKALKQAAETGVLSERRIDESVYRILKLKYKYTLADHPVPE
PDVGKINEQIGRVLDARLP
>lcl|NZ_CP026520.1_prot_WP_085981306.1_982 [gene=cmpA] [locus_tag=PC41400_RS05100] [protein=cortex morphogenetic protein CmpA] [protein_id=WP_085981306.1] [location=complement(1147045..1147191)] [gbkey=CDS]
MPQWLCNQLMRAFQNKNPRQIRLLNDSWFFYRTRRSPDELSGSFEKSR
>lcl|NZ_CP026520.1_prot_WP_042235925.1_983 [locus_tag=PC41400_RS05105] [protein=hypothetical protein] [protein_id=WP_042235925.1] [location=complement(1147288..1147653)] [gbkey=CDS]
MPTSVMKYAVLKLKDQLEFVEMPATHMYQLTALNGRLHKEIEKLTAHNVPELPVVIAECDNLEPVGTAYAVTGGLDYINK
LEASFASIQESAYPLVSLLTEIRALQAQLEQWYEEYEEEQQ
>lcl|NZ_CP026520.1_prot_WP_042235923.1_984 [locus_tag=PC41400_RS05110] [protein=SprT family protein] [protein_id=WP_042235923.1] [location=1147796..1148269] [gbkey=CDS]
MDDGQLQEWVEQISLQAFGWPFRHRARFNRRLRATGGRYFTKSHDIEISWHQWETYGRDDVEKIIKHELCHYHLHLQRRG
YQHRDADFKLLLKQVGGTRFCRALPQRKEPASYKYKLKCVACGTEYYRKRKMDVRKYSCGKCRGKLELYMLDFPKRS
>lcl|NZ_CP026520.1_prot_WP_042235920.1_985 [locus_tag=PC41400_RS05140] [protein=alpha/beta hydrolase] [protein_id=WP_042235920.1] [location=1149343..1150956] [gbkey=CDS]
MIQKVRARMIPLLGLSAVLACSSVVQAAPPPAPALHDDNPSGHFGDWYTGAVPPQASNDKPVILFVQGLHSNYNTWLKTD
GYYEAAYKAGYRTAFVQLKDADGTGGDMWTNGPKLAEVIKKVAQYYGVSQINIIAHSKGGIDTQAALLHYGAAPYVNVVH
QLSTPNKGSELADMAYSDWTSWLAEILGKKDDAVYSLQTSYMANFRTQTDSKPGVSATKTYMSGGEGDDGWFTAYWFAHA
ILPGEDDGAVSVASALGLPYGIKSFTKNISHSAMTTASQTWSLVQPKLALGKKAVAAAALQEENLSAASQAAPAVSENSS
FIMKGGSLAKGTAKEIFPLESGLGSISLDVLTAKETTQVTLTSPSGKKIPLNPAKSAEKDDAVLFSDAVHHTLTLENPEK
GNWTLELSGEGDAYFMLARPEGRVKGQIKNARKQYKQGENAQIAVSFDGAGKLEDTKAKKAKLTRSGGKGKVKSLQQAGE
VNLQTAKNEVLLNFKAPTEPGVYNISFDINGVNERQEPFTRSVNYNFAVTDEAGKIE
>lcl|NZ_CP026520.1_prot_WP_042235917.1_986 [locus_tag=PC41400_RS05145] [protein=pyridoxamine 5'-phosphate oxidase family protein] [protein_id=WP_042235917.1] [location=1151063..1151482] [gbkey=CDS]
MSNLSLEQQIVSVLDNNEFCSFGTVENGKPRVRYMALFHDELTIYLATNRKTDKVEEIRDNPHVHILVGFDGKPSSDILQ
IEATGSVSTDESLREKVWRDDFKEWFDGPHDPEYVILKIVPSRIEYYSGKSEQQVWQAK
>lcl|NZ_CP026520.1_prot_WP_042235913.1_987 [locus_tag=PC41400_RS05150] [protein=ABC transporter permease] [protein_id=WP_042235913.1] [location=complement(1151741..1152370)] [gbkey=CDS]
MELFFQYISENWLDLLGLTYDHIVMVLLAILMALIVGVPLGILAARYRRLGSVILTVANIIQVFPSLALLVLLMIVFGLG
FKTVVIGLFLYSLMPIIRNTFVGLTQVNPAITEAGRGVGMSQLQLMWKVQLPLSLPFIATGVRVAAVIAIGVATIAPLIG
GGGLGREIYSGINMDNSARLYAGAIPAALLALLTDFVLGQVQKRMHVSR
>lcl|NZ_CP026520.1_prot_WP_042235912.1_988 [locus_tag=PC41400_RS05155] [protein=glycine betaine ABC transporter substrate-binding protein] [protein_id=WP_042235912.1] [location=complement(1152403..1153302)] [gbkey=CDS]
MKRVRRRLLAASLLAAVVVSASGCGRGDTILIGTQTYSEPKILAQMYKLLIEDRTDLKVDVLPDLASSGVIIDAMKKGDI
QMSTLYTGELLNKHFPIEDKRRSREEVLAEAQAGFDKYFNFKWFNPYGFENTYAFTVRKELADEKGWKKISDIIPEAGSI
KLGVDTTWLERESDGYKAFAKHYGLQFGDTLPMEVALVYSAVASKKVDIVLAYSTDARLKEYNLVTLEDDKQFFPPYDAS
PVVKKELLADNPELNDIIGLLIGKIDAKTMIDLNYEVDVKKRNEKKVAEDYLKQIGLLK
>lcl|NZ_CP026520.1_prot_WP_042235909.1_989 [locus_tag=PC41400_RS05160] [protein=ABC transporter permease] [protein_id=WP_042235909.1] [location=complement(1153394..1154062)] [gbkey=CDS]
MDFLKSYGQFLSDRMPDILVAFREHLIISFLSIAFGCLVAIPIGVYLAGNRIKWINSLVFALSNLLQTVPSLALLAILIP
LIGVGMKPAIIALFLYSLMPIMRNTFTGFQAVNPHLIEAARGMGYSSIQRLVRIQLPLALPYIMSGVRVTTVYIISWATL
ATLIGAGGLGQLIVAGMGVNKKELIFTGAILAVILALIADFVLGALEKWLTSRTKTSGAKTA
>lcl|NZ_CP026520.1_prot_WP_042235906.1_990 [locus_tag=PC41400_RS05165] [protein=betaine/proline/choline family ABC transporter ATP-binding protein] [protein_id=WP_042235906.1] [location=complement(1154063..1155274)] [gbkey=CDS]
MIRLEGVIKRYDDGFVALKNLNLEFAEGQLHVLIGPSGCGKSTTMKLINRLINPSEGKVTIKGKDISQMDPVELRRNIGY
VIQNIGLFPHMTIAKNVAVVPKLLKWDQQRIDTRVDELLRLVSLDPDTFRNRYPSELSGGQQQRIGVIRALAAEPDIILM
DEPFSALDPISREQLQDELVRLQQDLKKTIIFVTHDMDEAVKIADNIVLMKDGEIVQHGNPEHILRYPANEFVKSFVGKK
RLQADTGFEEIPTVDEVMAEHPITAYPTRGLAEAIKIMEKNRVDSLLIVDRQNRLIGTTSIFQVLDKYRQENITLADIAK
PFHHTVETGTSLSVAIQLMNEHNLPYIPVVRENNTFAGLVTKGSVVRHIADVYTPAESDEFAETELPASAEDGKQTPVSS
GGA
>lcl|NZ_CP026520.1_prot_WP_042235903.1_991 [locus_tag=PC41400_RS05170] [protein=DUF2179 domain-containing protein] [protein_id=WP_042235903.1] [location=complement(1155374..1155895)] [gbkey=CDS]
MISAQTILTIIVINIAYVSLSTIRMILVMKARRLLAASISTVEIFIYLMGLNIVLKNIDDPVNIIAYCVGWGAGVWLGIR
IEGWLALGYTTLQIVVDYTETTLPDTLRTKGYGVTHWIAEGRDGPRLVMQVLAKRSLEKKLLSTLHELAPKAFVISYEPR
FFRGGFWTRRLNP
>lcl|NZ_CP026520.1_prot_WP_042235964.1_992 [locus_tag=PC41400_RS05205] [protein=ABC transporter substrate-binding protein] [protein_id=WP_042235964.1] [location=1162504..1163910] [gbkey=CDS]
MADGGRKKWSVTAVLLSSALLFGCSGQSVKTTPGPTQTGKDEGNSPAAKKTITITYRDDGIGEQGALFKWIKEVAASYPD
KSVEIKPTPIQASEGDYFAKIALALKSKDTAPDIVTEDTFILNSDASAGYLEPLDDRLKGWEDWTNGSFIEAMKKGVTAS
DGKVYGVPYNTDSRGLWYNKEIFKKAGLPENWQPKNWDEVLNAARTIKEKAPDVVPIWMNMGKATGEATSMQTYEMLLYG
TGERLYDDASGKWIVKSQGILDALGFIEKVTKEKLGPPLSKVLNGQAGNTATREYLPKGKLAISLDGSWITGNYLEGGAA
PWPAYKDVLGFAPMPTSKGQAPGSITLAGGWALSIPSNAQHKDEAWNFIKYALNKENTQKLVIASGNITVRADVAKDPGY
TKMPFNELATEYLKNAEFRPAQDKYPEVSTQIQTMVESVATGTAPAAALNKYTQDVTRIAGPEHVTEK
>lcl|NZ_CP026520.1_prot_WP_042235961.1_993 [locus_tag=PC41400_RS05210] [protein=sugar ABC transporter permease] [protein_id=WP_042235961.1] [location=1164073..1164963] [gbkey=CDS]
MSSMAVQNLYRKKRTSSWFYFLLPSIAIMLVFFIYPILLTFYYSFTNLALTGEAAKELKFTGFDNYVRMFQDPTVRISIW
NTLVFLIGSAVIGQQVLGFLIALLMKHKNKHFRRVIGTIVLAGWVTPEIVCALCLYSFFGDEGTLNAILAFFGIPVVAWL
YTVPMITIILANIWHGTAFSMLVFQAALDDVPAEIEEAAVVDGASRWQILTRIILPLIKGTLTTNMMLVTLQTLGVFGLI
YAMTGGGPGTSTTTLPIFMYNQAFVNYQLGYGTAISLLLLLLGIILSLFYIRAMKE
>lcl|NZ_CP026520.1_prot_WP_042235958.1_994 [locus_tag=PC41400_RS05215] [protein=carbohydrate ABC transporter permease] [protein_id=WP_042235958.1] [location=1164985..1165818] [gbkey=CDS]
MNTSQRKWMYRILPYTILVFIGICFILPLLWVLVASVDSNAMQSLKVPERLTATNYAEVIANPENQRAYMIGLIMSLGQA
VLVVVLALLAAYPLSRYQLKYKKTFMLTVLFMTSLPITAVMVPVYQLFLTAKLYDNLFGVILFFAASSMPYGIWMLKNFM
DSVPSELEEAAWIDGASVFTGIRKVVAPLMVPGICAVAIFTFSGSWGNFFVPYILLQSPENFPVSLKLYQFFGQYGMADY
GKLAAFSVLYAVPSVILYVLSQRFMSKGFGLQGGTKG
>lcl|NZ_CP026520.1_prot_WP_042235956.1_995 [locus_tag=PC41400_RS05220] [protein=trehalase family glycosidase] [protein_id=WP_042235956.1] [location=1166139..1167890] [gbkey=CDS]
MNFDLSVVPFSRFGSYFALSRLPQSDKRSEGLYVRMIRGGDEQAGVLFGITPIDPKSGETLADYRVEASPVRVRLESGAG
WMEWCIPEPDCIRVRGEGIGLKLTAELSAYDYALPYRERGWELNSFSREVRILLSPLLGHFEPDVVWGGLRAKHIAFVIV
PGGESQSFDLSLEEYRTVPGQRDGYSYEEAEAMAGQDWEQWLSRTIRTPEDWEEARTLAAYITWSCVVRADGYLSRPAMY
MSKNWMTNIWSWDHCFNAMALAGTNPDLAWDQFAIFFDRQDESGMLPDFLNDKYALWNCCKPPIHGWTLNWMMENSSAVT
DERLREIYEPLGKWTTWWFRYRDDDGDGIPQYNHGNDSGWDNSTAFLTDIPVESPDLSAFLVIQMEVLAAAAQRLGLDEE
SERWSARANELLQKLIGHFWDEKEGRFTAMRSGSHERSAGDSLLSFVPIVLGSRLPGPIREKVIADLKQPDRFLTGNGLA
TESTSSPYYAADGYWRGPIWAPSTMLIVEGLRGAGDEEFASELARKFCRMAAKSGMAENFDAVTGEGLRDRAFTWTSSVF
LLLAGRCREEEMVSGASAGSPEQ
>lcl|NZ_CP026520.1_prot_WP_129112455.1_996 [locus_tag=PC41400_RS05225] [protein=hypothetical protein] [protein_id=WP_129112455.1] [location=1167912..1168118] [gbkey=CDS]
MTVKTGTGSDTGTKTRGKQSKMQARQEANQERKHTNQEANQPSKPTKKTKQTNQENQANQPRKPSNKA
>lcl|NZ_CP026520.1_prot_WP_042235954.1_997 [locus_tag=PC41400_RS05230] [protein=SDR family NAD(P)-dependent oxidoreductase] [protein_id=WP_042235954.1] [location=1168220..1168999] [gbkey=CDS]
MTTNTSPLDLTGKTALVTGAASGIGLVAAQTLAANGAKVMLLDMNETGGREAERFLTEQGHFARYIRCDVTRSEDCQAAV
EAAVRETGRIDIVFNNAGVIRRKTVVELTEEQWDLVIDVSLKGTFLLSKYAIPVMEKNGGGSIINTGSGWGLKGGDQAAA
YCAAKAGVVNLTRGMAIDHGPAGIRVNCVCPGDTDTPLLRDEAKQLSRDEASFLVASAERPLNRLGTPQDIANAVLFLAS
GMSAWVTGSVLVVDGGGLA
>lcl|NZ_CP026520.1_prot_WP_042235951.1_998 [gene=gcvPA] [locus_tag=PC41400_RS05235] [protein=aminomethyl-transferring glycine dehydrogenase subunit GcvPA] [protein_id=WP_042235951.1] [location=1169424..1170827] [gbkey=CDS]
MATRQSSSHPYIPNTVSDVQKDMLQVIGAASIEELYRMVPESLRYRDTLQIGPALNEYELQRHIETLLNRNLHARERLNF
LGAGCWQHYVPAVCDEISQRSEFVTAYAGEPYEDHGRFQALFEYQSLLAELVNMDVVNVPTFDWAQAASTALRMASRMNG
RSEVLLPATTHPDRLLIIRNYLSPQIRCTLVGCDPATGLLDLGDLQSKLSSDTAAVYIENPSYLGHIEVQGEAIGKLAKE
AGAVFAVGVDPISLGVLKPPSRYGADLVCGDLQPLGIRMNYGGGHAGFIASRDEEAYVMEYPSRLFGIAPTSVEGEYGFG
DVAYERTSFAKREKGKESVGTQTALWGITAGVYLALMGPQGMLEVGQTIVQQAEYAKKRLSEIEGVSMKFSASTFKEFVV
DFNSTGKSVAEINKLLLEENIFGGKDLSAEFPEWGQCALYCVTEIHTKQDIDTLTEALKKVLGRAGR
>lcl|NZ_CP026520.1_prot_WP_042235948.1_999 [gene=gcvPB] [locus_tag=PC41400_RS05240] [protein=aminomethyl-transferring glycine dehydrogenase subunit GcvPB] [protein_id=WP_042235948.1] [location=1170992..1172566] [gbkey=CDS]
MIKRIERNSKVRKFHQAKWDEPVLFELHRPGERGILVPAAEPEVEAEVGDGLSALPASMRRETAPALPEIGQAKVLRHYL
RLSQETLGSDFNVEIGQGTCTMKYIPRINEMLIRSPKMMDLHPLQDPDTVQGMLEIYHGLDLAMREISGMDRFSFQPSSG
TQALFAMASIVRAYHDANGEGEKRDEIITTIFSHPSQAATAAVKGYKIITLYPDKDGFPDLEALKNAVSERTAGFVVANP
EDTGIFNPRIREFTDVVHEAGGICFYDQANANGLLGIARAKEAGFDMCFFNLHKTFAAPHMCGGPATGALGVTAQLAPFL
PGPVVEYDGSRYSLSGETPESIGKVRMFHGVAQTVLRSYAWIRALGPEGLLDVAKTAVLNNNYLYANILKIRGASAPYVE
GGRLEQVRYSWEQLTKETGVTTEDVQRRMCDFGLHYWTSHHPYIVPQPFTLEPTESYSKEDLDEYLDALRHISQEAYENP
EIVKSAPHNSTVHRIDESSFDDPKEWCITWRAYLKKTQPAGAAR
>lcl|NZ_CP026520.1_prot_WP_241688795.1_1000 [locus_tag=PC41400_RS05245] [protein=sensor histidine kinase] [protein_id=WP_241688795.1] [location=1172735..1174441] [gbkey=CDS]
MLFRHLGLLPIRLRRSLQGKLVLLILLVAIIPLLSLGYVLYTKSSQTINEQFGRYGENTVSQLQVQIDSTLAQMKYTVGD
ILSYLLDPNFTVLHEEIPTTYLGFKEEQQLEQYLKAHKTLDTKGIFILTKSGYYYGERIANTDLVEESPWFGEERTGYRF
SIYEPKQYSGTSNVPGEKVIGLLFQIHNQVGALEDAKVLIEVKAEKLLDMFHEYEQDTGAFLTISDQGRILYQTAKAYTK
RDDDVVWTKQSEETDWKIEVRTPYHQFYQSSLLIRTFTFTAVSLSILLAFVLAYFSSLRFLSRIKKMKDSIRDVSLGRLD
TRIVVDSEDEIGRLGRSFNHMVVQLQDLFGEVKRVERQKKEAELRALHHQINPHLLINTLASIQWKARIAGNAEVQQMIL
HLTKVLDGNLNYTRELVTLKEELDVIHHYMNLQEYRFGHVFSYRIDTGEIPLDQVRIPRMTLQPLVENIFFHAFEDGAGT
IEIKVREQGKAIRLSVCDDGKGMEPEAARQLLSDPGDAVGRKKRSIGIYNVDQRIKLHFGDKYGLDVESEPGKGTCIGIK
LPKEEETG
>lcl|NZ_CP026520.1_prot_PC41400_RS05250_1001 [locus_tag=PC41400_RS05250] [protein=response regulator] [pseudo=true] [partial=3'] [location=1174438..>1174821] [gbkey=CDS]
MSDTTIRVLIADDEVIIRRGLISTVPWERYGMEVVADAPNGTRAWEAFLEHEPEVVITDIVMPGMTGIELIRKIKEYAPQ
TKALLLSCHRDFEYAKQGIKYGASGYLLKTDFSDEELGDYLEGFRSEL
>lcl|NZ_CP026520.1_prot_WP_241688881.1_1002 [locus_tag=PC41400_RS05255] [protein=AraC family transcriptional regulator] [protein_id=WP_241688881.1] [location=1175317..1175997] [gbkey=CDS]
MPAHACGAPASAEAEAPGATATPGPATCAGRLAAGLASAGLCAAAALSGQHVLLLVPQGERQAWNARLCALKAADPQLRW
TEGAAFAGASGWLAEVRSLHGRIELETRHRVRIGEWPEPIAAAVRLIVEQADSQLSVAEAAHEVGLSRSHFSTLFKRVVG
ESFVAFTCRTKLGIASELLAASKLTVQDIADRVGMPDAKYFSKWFKRCTGQTPSHYRTERRMTDAR
>lcl|NZ_CP026520.1_prot_WP_042229903.1_1003 [locus_tag=PC41400_RS05260] [protein=sugar ABC transporter substrate-binding protein] [protein_id=WP_042229903.1] [location=1176245..1177513] [gbkey=CDS]
MKKTLVLALTGSMVTALALSGCSSKQGEAQGGGDSGSSGGKVTLRFATWDTGDLLKIQETIAKNFEAKHPNVKVQVEAYG
DGFDQKLAASIGANNAPDVMYMWDFPTYYKNLEPLDERMKSDATLKQDDFYPGLFNYVKIDETAYGIPAGFTTRVMYYNK
KLFDEAGVPQPKEGWTWDEFKSAVQKLTNPAKKQYGFGVRAENDPYDLQGIVWSNGGSFISDDGKTIEGYMNGKPAVEAV
GMIGDLVKAKTAVIVGGKNQQSGDDIFKAGKIAMWESGIWPLDGFKEAKVDFGTVSMPVFPGKPVKGVIAESAVSMSKNS
KHKEEAWEFIKYYASEEAIKLRTSDLPVRKSIVEGNKLDQDPVYKAFYTMLEASDNTPAFLLSPKWNEVNRNLTSAINAI
MLGQNAQDSMDTAVRDSQKFFK
>lcl|NZ_CP026520.1_prot_WP_042229901.1_1004 [locus_tag=PC41400_RS05265] [protein=sugar ABC transporter permease] [protein_id=WP_042229901.1] [location=1177540..1178502] [gbkey=CDS]
MQPNTLQLQKGVRRSIWRRGRLTPFLFILPWMIGFVAFTLGPLLFSLVISFYDWPIVGERTFVGFQNYVNMFTNDPLFWH
SLGVTIRFAAIFVPLNLAIALLLAVLLNQNVRGSGIFRTVFYLPSVISGVALAMIWAWVYSGEYGILNYFLSLIGITGPD
WLREPNWSMIAMVAASLWAQGSMMLIFLAGLKNIPGDLYEAAELDGAGAFKRFIRITLPMLGPTLLFNLITSIIAAFQQL
TLALTLTGGGPLNSTYFYAMYVYENAFKYFKMGYSAANAWVMFVIVLALTMMVFRSTSMFVYYEGELKKNKARKTKRKAL
>lcl|NZ_CP026520.1_prot_WP_009672023.1_1005 [locus_tag=PC41400_RS05270] [protein=carbohydrate ABC transporter permease] [protein_id=WP_009672023.1] [location=1178524..1179333] [gbkey=CDS]
MKKVLTYALLLFFSLLFLMPFFWLVTTALKSPEEVYMYPPKWIPSELHWENFAKAWTIQPFNVFLKNSLTVTVLCNIGQL
ISSSLVAYGFARFRFKGRDALFMVVLATMMIPWDVTMIPLYMEFNFLGWINTLKPLIVPSWFGSAFYIFLLRQFIMTIPT
ELEEAAKIDGAGDLKIFTKIIIPLMFPALVLVSVFNMLDTWNDYLGPLIFLNDASMYTLTLGLAQFKGMFGVDMTSVMAV
TTIICLPPLAVFFLAQRYIVEGISMTGIK
>lcl|NZ_CP026520.1_prot_WP_042229899.1_1006 [locus_tag=PC41400_RS05275] [protein=MFS transporter] [protein_id=WP_042229899.1] [location=1179483..1181039] [gbkey=CDS]
MFSFRSSNTSMTTGVRRFLATEALYGIGIGLYSLALNLHLLALGASGGQIASISAVGILIMGIGALPAGMLAGKIGRKPM
LLAGSMLVAAGAALFATGRSLPVWYAAQAVASAGFMCIETTEIQLLFRYCRSRREESRAFHGMFAVFTAFTGLGNLLGGI
LSRADASAAGHAPSLYASAVCIAVLSLLRLRLLPAERAAGGSGGASSMGDSGDGGRPGPAGADASGSGLRPGAEAAAGGA
SGNCDAAEAESRGRDTTDKGPGDSDRREAPEAPEAAASEACDAEEKNASRRGPAKPPVSRERRPAGRVSRLPQGFRRLLR
NRSLWLLASNACLIGFVFSFIGPFMNVYLKFRAAWGDDLISYVYTVNGLVLFAASLAAPKIMERWGAKVAFTGAFLGNAG
LALLLLIAQPAPVFAALLVLRSGSFSLLGNLVDSETMSSLREEERNLYAGMRAVFRCIGSSLSTMTAGFFMNAGDYHTPF
LIAGVLLIVNFMYFRSRVLPLIIKRNDNLRGEPQAGGA
>lcl|NZ_CP026520.1_prot_WP_042229897.1_1007 [locus_tag=PC41400_RS05280] [protein=glycoside hydrolase family 2 TIM barrel-domain containing protein] [protein_id=WP_042229897.1] [location=1181042..1184548] [gbkey=CDS]
MKELHLQRRDWENLDVLERDRLPARASFIPYGKEKEALTYAGERSDRMLLLNGTWAFRYTGSPEEAPAGFYEPEFDASAW
DRLPVPSCWQMHGYGAPHYTDLYYPFPVDPPRVPAANPTGSYLREFTLPDDWTDKDIVIRFHGVDSAYNVWLNGEYAGYS
EGSRLTAEFDLTPFIRSGRNVLAVQVYQWSAGSYLEDQDMWWLSGIFRDVVLTARPKLHLQNIRVNAGLDEACRHGLLRM
AVTVAEAAGCAARIGGQNGEHNGTQTGTQTGTQTGTQIGTQSDAQNRAHSDAQTGAQNETQNGSQTRTGSATDTASRPFT
LAWKLQDQKGNLIDSGEENGIAGSGKTDGIAGTDKERSFTIEAEIKEPRKWSAETPYLYDLIVTLLDEEGEVLETTALKT
GFRRIEVKGGLMLVNGKAIRLRGVNRHDHHPDTGRTVTPETMEKDIRMMKRHNINAVRTAHYPNDPRFYDLCDRYGLYVM
DETDLETHGFELIGDANRLSADPAWKHAYVDRMRRMVMRDENHPSIIMWSLGNESGFGCNHEAMYEWCRSYDPSRLVHYE
GDREGKACDVFSTMYSSPEKMRGFGEMKELDKPHILCEYAHAMGNGPGGLQHYEELFRTYPRLQGGFVWEWIDHGLRRKK
EDGREYFAYGGDYGDQPNNGNFCIDGLVTPDRMPSPGLMEYKKTIEPISVEIAGWEPGNNEGAAELIIRNHYDFISLQNF
RCTWKLEADGRLVGGGAAELPDLAAGEEGTLHIPLSVPAADEGGPVEHWLTVRIVLASPASWAEEGHEIAWAQFPLHACS
EEAAGAAINSDAESPAAASEGQMAASQAYGGEGSAADNAAVPAGVPGAALGHQEPAPAAAAAAIRVEEADRLLTLTGGDF
ALVFDRGSGRIASWRHAGTPVIAEGGGPRLALWRAPIDNDMYVVADWRKACVHMAADDVRGSGWERLADGSVRVSVSIRT
APPVYAWGFSCTWVYTVRPDGTVDLTVQGEPDGTPPDMLPRIGLRMELPGSMDRVTWYGLGPGESYPDSRQAARMGLYDA
AVDDLRFPYVKPQESGNRSDVRWAYLRGVQGSGLLAAGQPSFDFGASRYTPEDLEAASHECDLVPRDYVTLHLDYKQNGL
GSNSCGPKQLPPHVLEPQAFRFAVRLGPVSGDAHPEQTAAALRSGGIS
>lcl|NZ_CP026520.1_prot_WP_042229895.1_1008 [locus_tag=PC41400_RS05335] [protein=LacI family DNA-binding transcriptional regulator] [protein_id=WP_042229895.1] [location=1187018..1188040] [gbkey=CDS]
MAKTNKEITIKDVAKQAGVSVATVSRVMNGRDRVSEATRKKILKVVEEMNFVPNTMAASMVNKKTNMLSVVVPEIQNPFY
TAVIGGMVEVAKKEGFTTLVVATNSDEAEEEEFFESFLGKNVDGIILIGTHKDAEFYRNIRKPTILVDRYINDCGHDGVL
IDNFRGAYEATKHFVDYGHERIAIIDGAHDFNDGKDRYWGYRQAMHESGLTPDPAYHKQGSWLEEDGYRFTTELMQSEQP
PTAIFAANNVICTGAIKAIRDLNLRIGEDISLIGFDENELAQFVQPRVTVVSRPTREMGIQAAEMLIQKIKGKPKEQTKL
KKMILDVELKKFGSVKRLKT
>lcl|NZ_CP026520.1_prot_WP_042229894.1_1009 [locus_tag=PC41400_RS05340] [protein=ribokinase] [protein_id=WP_042229894.1] [location=1188056..1188964] [gbkey=CDS]
MAKILVVGSMNMDIVTQVERHPIPGETIQGNETRFFIGGKGANQAVAASRSGVSVAMAGGLGVDSFGADIRRGLAEDNII
LDYVADQPCMTGLALITVDGAGENNIILSSGANGRYGAEEIDQLDLSGFDVILLQNEISSAVNQRVLDKAAEKHIPVFMN
PAPIDGFDRSQLPKVAFLILNEVEAEVLSGVRIEGSSQALEAGKKLLKDGIPHLIITLGSKGSIYMDQSGSQITFPAFSV
DAVDTTAAGDTFIGAFAAKYVTGAELHDCLRYATAASALAVAQPGAQSSIPTDDQVQSFLTA
>lcl|NZ_CP026520.1_prot_WP_042229892.1_1010 [locus_tag=PC41400_RS05345] [protein=sugar ABC transporter ATP-binding protein] [protein_id=WP_042229892.1] [location=1189119..1190630] [gbkey=CDS]
MRHPVLKMEGISKEFPGVKALSDVSFELYPGEVHALMGENGAGKSTLMKILSGVYSPTKGTIRLKGKEVTFGNPLDAQRQ
GVSIIHQEFNLFSNLTAAENIFIDRPEMVGRFGRIQWSKMYDEAQRLVDSIGGGEIDVRQEVRHLGVHSQQVIEIAKALS
FQADVLIMDEPSAALPENEVKKMFDVVNRLRAQGVAIVYVSHRMKEIFEIADKVTVLRDGAKIDTRGIEEVTEQNLIQMM
VGKEVGQLYPVREQKHGQEVVLKVEQLSLDSAHRVSFELRKGEIVGLFGVPGSGTHTIAERLFGLRRGTGDIIVHGAAVK
INSPSQAKAKKIAYVPPDRHRQGIIKPMSIRQNLSIPMLPQLSKYTVLDEARIKQMSGEYMEKLRIKAPSDGQSVDFLSG
GNQQKVVIGKWLAANPDILILEEPTRGVDVGAKAEIYNIIHQLAQEGLSILLISSEMPEVIGLSDRILVLHKGTIVHEFA
HGEADQEQLLQRASHPLPEGGLL
>lcl|NZ_CP026520.1_prot_WP_042229890.1_1011 [locus_tag=PC41400_RS05350] [protein=ABC transporter permease] [protein_id=WP_042229890.1] [location=1190627..1191589] [gbkey=CDS]
MKRLLNVHEAPIILFTILVTLLFSIISPDFLQFENIRLIVDQNISMFIVAIGMTMVILLGGMDLSVGSVLAVTATSVGLF
LGHGIHPWLAAAMGICIGVLCGMVNGYFIAVRKIPDIIVTLATMYIFRGIAVGISGGTWMTNFPEGFRFFEQGYLLGISF
PLLTAVILIAIFIYLMKFTRVGRRIYSIGGNGPAAKLAGISIARTKFKVYVLTGLLVGIASVIFAAKVGSVQASTAGSGL
AFEVMAAVLIGGGSIFGGVGTIAGTMLGVLFMGIVKNGLIISKVSPFWVDAATGFLIILAIVINTLQRIRQNKGKEGDLV
>lcl|NZ_CP026520.1_prot_WP_042229889.1_1012 [locus_tag=PC41400_RS05355] [protein=ABC transporter permease] [protein_id=WP_042229889.1] [location=1191586..1192530] [gbkey=CDS]
MSAWTSRFPLRGNKLIVQMLLLLVIIVLFSFLSPYFLTVTNFMNVLNQMVEVGLIAIPMTMIIISGAMDLSVGSILAFSA
VCMGVAYERGIDIWISVIIGLLAGLLCGAVNGYLIARHRMQAIVVTIGMLVMLRGLVYVINQGRPISGFPDAFYFMGQQY
IGVVPVNAIFLVVMFLLANFVMKKTRFSTYVYGIGNNEEVVRYSGISVVRIRFILFMLSGLFSALAGIFLVSRLASAEAT
SGTNIELDVITATLIGGTHIFGGRGSLSGTFVGLLIIVFLRNGLNLLGVSVLYQAVILGALLLLAVGNKKFNGE
>lcl|NZ_CP026520.1_prot_WP_042229887.1_1013 [locus_tag=PC41400_RS05360] [protein=autoinducer 2 ABC transporter substrate-binding protein] [protein_id=WP_042229887.1] [location=1192536..1193567] [gbkey=CDS]
MKKLALTLLTGVLALNLAACGSTGGNQTDNGGKPNQGASGSKTYFMNPKSIGPAYWAAAEKGAKKAAADLKVEVLFNAPT
EADSSKQINMIQDMLTRKVSGIGVSPNDAKAVGPVFKKAAGQGVNIVTWDSDAPGTDRQYYVAPATDQEVGELLAKTIGE
EIGGKGQVAFMVAGLGSQNQIAKSNAAKAYFTAKYPDIKVVTTVASDDDQQKAYANAQNLLQTYPDLKGIIGFAGGEPTA
AAEVVEQAVKAGTLKKNQVAITGIAVPSVVKNYIKNGTIKTDIIWDPGKLAYVTVYIMDQLAQGKEIKDGMEIPTIGTVK
VDGQSVFIGTLEVTNDNVDSFDF
>lcl|NZ_CP026520.1_prot_WP_042229886.1_1014 [locus_tag=PC41400_RS05365] [protein=ureidoglycolate lyase] [protein_id=WP_042229886.1] [location=1193596..1194105] [gbkey=CDS]
MEYKVRIEELTEEAFAPYGKVVDLPAIDPSKSGDGWDCWSYVQMLDVWEPIGFGLVVTKRREFAVTAMERHVSREELLLT
FDREIVQPVALCMDIDDPDESPDPSTVKCFRIKPGQAIVIHRGVWHSPAYPAADDARYMFGIEKKKDKFGDEMINPWVDF
IDGSTVRFE
>lcl|NZ_CP026520.1_prot_WP_042229885.1_1015 [locus_tag=PC41400_RS05370] [protein=ArgE/DapE family deacylase] [protein_id=WP_042229885.1] [location=1194195..1195505] [gbkey=CDS]
MNQEQRVMEWIENHQDEIIHFLQQLIQIPSVNPWFFEEPAPSKEKDLQQFIAGKLEKLGADIEIWEPVADELKQYEGMAG
YYPGRDFTGRPNLAATFGKGSEGKSLLLFGHIDVVMAGSKWTVDPFEGQVIDGKMYGRGTVDMKGGVAAMIMAVEAILRS
GFKLKGPVIVGTVVDEEAGGMGALDFIHHGYRADACILTEPTSLTIAPLCRGILWGKIKIQGRSGHIEMPQADWRDGGAV
DAIKKARYILEHLDRLNEKWAVSKTHPYLSIPCQVHVAQLNAGEYPTSFANEAEIVFNAQYLPSERDEKLVGGKVKKELT
DFLREVSESDPWLCEHMPEIEWLVDADCAETDVAHPFVQTCVQSLKTIGEEGKIEGLGFHTDMGWPVNVGIPTVNFGPGD
PSLAHHSDEFTPVDEVIQAVKMIAKTIVDWCEIEEA
>lcl|NZ_CP026520.1_prot_WP_084706592.1_1016 [locus_tag=PC41400_RS05375] [protein=glutamine amidotransferase] [protein_id=WP_084706592.1] [location=1195508..1196305] [gbkey=CDS]
MGSRRPAGVGPGGIKNTEVLELMKILFVGESWVVHMIHTKGYDSFTSTKYEEGATYLLSCLRQEGIDVTYMPAHEVQVRF
PQSLEELKTYDAIVLSDVGANTFLLQNPTFYQMQIIPNALELVKQYVAEGGGLLMVGGYLTFMGIEGKANYRNTILADVL
PVVMMDKDDRVEMPSGFSPVVSQPHPLVNDLGEWPRLLGYNKLSAKPGAEELLSHEGDAILTVGTYGQGKTAAFASDCSP
HWGSLEFMNWEHYSAFWSKLIKYLK
>lcl|NZ_CP026520.1_prot_WP_042229883.1_1017 [locus_tag=PC41400_RS05380] [protein=sugar kinase] [protein_id=WP_042229883.1] [location=1196801..1197775] [gbkey=CDS]
MEQGKEQPPIYPDVVTFGESMALLMPDSGKGIEYSPQLTTSFGGAESNVAIGLARLGHRVGWFGRLGKDPFGRMVLKKIR
GEGVDVSRAVLASEAPTGLMLREAVSGKASVYYYRKNSAMSLMKPGDLEEDYIAGAKILHVTGITCALSESCKEAVIAAV
RTAKRHGVKVCFDPNLRLKLWGIGEARETLLALAEEVDYFLPGLDELQLLYETEDFDEIVGKLSKLRAVTVLKGGDNETF
LIEGGRVSSVPFVKAERVVDTIGAGDGFCAGFLSGILQGKSHLEAVELGSLLGSLVVQAPGDWEGLPTREEVDAVLQGSR
HIER
>lcl|NZ_CP026520.1_prot_WP_042229882.1_1018 [locus_tag=PC41400_RS05385] [protein=bifunctional 2-keto-4-hydroxyglutarate aldolase/2-keto-3-deoxy-6-phosphogluconate aldolase] [protein_id=WP_042229882.1] [location=1197907..1198578] [gbkey=CDS]
MKKIQLIQKITEQGVVAVLRGKTPDEVVEAAEQAIEGGIKVIEVTMTVPFALQAIEKLAKTYSSTAQDASKFAIIGVGTV
LDPETARAAILSGSEFVVGPSLNPATVALCNRYRVPVMPGTMTVKEIQEALELGVDIVKLFPGNLYEPSVIKAIKGPLPQ
ANIMPTGGVSLDNLADWIKAGAVAVGIGSDLTTEASKTGDFSLIAKKAAQYTQAFKEAKAALS
>lcl|NZ_CP026520.1_prot_WP_042229881.1_1019 [locus_tag=PC41400_RS05400] [protein=hypothetical protein] [protein_id=WP_042229881.1] [location=1199174..1199665] [gbkey=CDS]
MRRNKWIGLSGGLVILVMAVAGAASLWLPSAELKPKKVVLNWDGRIIEVPQADSRYQRLVDDAEAVIESTAKGPCKCTAE
LTGDVPDKPGLLLMLPPGQSVRIRKELPLASLVNDVRRLYVSFQDGGDSHRMELDGGVYEVSSESLNALKKTAAEAAGNF
REE
>lcl|NZ_CP026520.1_prot_WP_042229880.1_1020 [locus_tag=PC41400_RS05405] [protein=alpha/beta fold hydrolase] [protein_id=WP_042229880.1] [location=1199828..1200589] [gbkey=CDS]
MTEKRRSRGRKWLLALVIFILAAGIAAAFYLRPYGPDRTAVESMRSGGGVTVTDSKDVTVYEGAEPKEPGVIFYPGGLVK
PESYAPLARELARRGHRTVVVKMPLNLAVLGGGRASAVLEAHPGESFVIGGHSLGGVMAARYAAEHPEGLRGVFFLASYA
DDKGSLRDKGLPVLSVTGSEDGVLNRDSFEKNKKNLPEDAKLLTLPGGNHGQFGSYGPQKGDRAASISPEEQTRRTAEAI
TDWMTGIREKGKN
>lcl|NZ_CP026520.1_prot_WP_042229879.1_1021 [locus_tag=PC41400_RS05410] [protein=GtrA family protein] [protein_id=WP_042229879.1] [location=1200593..1201045] [gbkey=CDS]
MTNDNNGQKKRVNPEMLRLLRFGAVGVGNTAVDWAVFWLLFSGLHLHYGAAQTVSYLCGTLNSYLWNRKWTFASKEPVRV
SEALKFLLVNGLCLVLSYLLLYVCSDVLGWNMLFSKLTVTVLLLLVNYAASKWWVFAPGEKRKDGESLEP
>lcl|NZ_CP026520.1_prot_WP_042229878.1_1022 [locus_tag=PC41400_RS05415] [protein=MFS transporter] [protein_id=WP_042229878.1] [location=1201232..1202515] [gbkey=CDS]
MQAKTRSWNRFTPTRKHLWIATLEGAPATVVQTLLNGPFLTGYLLYLGATSAQIGLVLAMTTIVNVGQILMAFFMQRWKN
RKLQILIFGGLHRILWAATGLIPLLLDRDLWIPAYIALYTSAFLSNAFGGVVWSTLMADMVPSSVRARYFGIRNMTLNAL
GALCLFAGGQILDKLGNEAGFEVLYLVIGTAAVLNVIAYLFYPNPPFEPSAESGLLPMLTRPMKDTSYIRATAFLSFWLF
LQGIAVPLFSYVMLKLLQVSYETISVLTVLQTAVMMISFYVWGNLNARWSNRRLLFWTLPLLSLSCLLWGMLAWLPVLPV
LIAVHILLGAGTGGFNQLAFNFIIGDTPRGERPMFIAMYNTLTGLAAFLGPLLGGVLYEKAEGLPLWVQQYGIAAGTGTI
MLVLALTLGRKVLLERQSRRTKTKDAG
>lcl|NZ_CP026520.1_prot_WP_042229876.1_1023 [locus_tag=PC41400_RS05420] [protein=serine/threonine protein kinase] [protein_id=WP_042229876.1] [location=1202667..1203302] [gbkey=CDS]
MEAWIRKIERELLPGLDFKSEKPYDPVVVHKLPAPWQLVGAGNYAIVVSHPDMPGLVVKVYAPGRDGWEDEVEVYRKLGE
HPAYSRCHYAKKPYLILKRMNGRTLYDCIRSGIRIPERVILDIEEACDYARRLGLVPQDLHAKNVMMEDGRGKVLDVSDF
LGDTPDTMWEDMKKAYYRLYLPLLSKRPIPMPDTLLNGVRKSYRWLRRNSS
>lcl|NZ_CP026520.1_prot_WP_042229875.1_1024 [locus_tag=PC41400_RS05425] [protein=ABC transporter permease] [protein_id=WP_042229875.1] [location=complement(1203489..1204418)] [gbkey=CDS]
MLNLIRNENMKIYRRLTTWIMVILLVGLVGLISFLIHYDSRGMNSADWRQAQTQRIEQLEKTKADFPDNAADLDQQIKLA
QEYLNRDMSPNQAAVWQAVGSMSSLIAVITVFTVVVAADAMSGEFSSGTIKLLLIRPVSRSKIVFAKYAAGFQFSIFMLL
LLFTSALVINLVLYGGAGTSPTVLELNSMGAIVEKNVIIDMLSTYGFKCVELIMIVTLAFMISTLFRSNSLAIGISLCIM
FLGNTVSMILSRYDWSKYLLFSNMDLTRYLHGQPLVPGMTMTFSILVLAAYFIVFNVLTWTVFRRRDVM
>lcl|NZ_CP026520.1_prot_WP_042229873.1_1025 [locus_tag=PC41400_RS05430] [protein=ABC transporter ATP-binding protein] [protein_id=WP_042229873.1] [location=complement(1204411..1205340)] [gbkey=CDS]
MNNQPVLEIRGVNKTIRGKKIIQDLSFEVYPGEVFGFLGPNGAGKTTLIRMIVGLMKMNKGDIVIHGQSIQKSFEGAIKH
VGAIVENPEMYKFLSGQNNLVHYARMFPGGIPQSRIDEVVELVGLKNRIKEKVKTYSLGMRQRLGVAQALLHRPSLLILD
EPTNGLDPAGIRELRDYLRRLAREEGIAIVVSSHLLSEMELMCDRVAIIQQGKLVDVQDLRGEGAAPKETSYFFEVNDPA
TAQAVVQEIDPHLHVTPHPNGIELTIGRERAADIGALLIQAGLRLYAMKPLTQSLEDKFLQITGSDQIA
>lcl|NZ_CP026520.1_prot_WP_042229872.1_1026 [locus_tag=PC41400_RS05435] [protein=substrate-binding domain-containing protein] [protein_id=WP_042229872.1] [location=1205608..1206618] [gbkey=CDS]
MKKWAPSKWTVGLAVLFALFVLLLGRFAYTNAEINGIVRDIEDRGRTPASGGKTKHVVMISQELDNPFWRTIEKGARDAA
EQYNIKLEYSGPYRNNIEEQAELFGKAIAGKPDGILIQGIRSEAYNKQIDKAVEQGIAVLTVDTDSPDSRRLSYVGTDNV
ESGRQLGELVAKAYEGRSASIGMVIGSYEAGNQQQRLEGFRSVVDAEPNLKIVEVGVSNISRIQATQAAEHMLRSREGIN
VMVGTSALDALGILQASKSLNKQEQVRIFGFDDLTETREALARKEIEATVIQKPYTMGFRSVEMLNRYFQGETLPGFSFT
RIDILNPQNVGTAVAP
>lcl|NZ_CP026520.1_prot_WP_042229871.1_1027 [locus_tag=PC41400_RS05440] [protein=sensor histidine kinase] [protein_id=WP_042229871.1] [location=1206615..1208183] [gbkey=CDS]
MSIRMKLFLFIPALVILMNSVTFFIYQSGRTIQESYHLTMDRILLYKEIAHKTEESLRSLSSYLINLNGEGRGAFVALQE
ELSSLQGKLESDARSEEDSFARRNYANMLGTYLAAADSVLAAVESKDMQRYLQRYEEADKTAGYIKAEHQNLVDMELSFY
QPLYKQMLLKTGEMNMLGQTLVAVNMLLSLVFAIWLSQSITKPIGRLVHAARQMSKGHLDTAPPPVQNEDEIGILTLAFG
HMQVNLKELIAKNQANMEKDRLVKELEIKALQSQINPHFLFNTLNVLSKLALLEGAEKTSDLTVSVSNLFRYNLRKLDQP
VSLREEVDHAREYFAIQQSRFRDRISFETDIDEGALEQQIPCLTLQPLLENAFMHGIESMEEGAFIRLGIMKAGGNVRVT
VADNGAGMNEETRLHLLRYGSASGDGAVPAAREGISAGEEEGVRRIEAPSGYDFSRTERKPSTGLGTQNVFRRLQLFYGT
DGLIEIESAPGKGTTVVLTLPEMTGQRAGIRGDAYVPLADRG
>lcl|NZ_CP026520.1_prot_WP_053228708.1_1028 [locus_tag=PC41400_RS05445] [protein=response regulator] [protein_id=WP_053228708.1] [location=1208158..1209807] [gbkey=CDS]
MYRLLIADDEALEREGLEWMISRSMPGVFEVIHAENGRTAIQRAEQHRPDIVLMDIKMPGIEGLEALRRIKEIHPRMKLV
LVTAYDYFSYAKQALEIGVRDYMLKPAKREQMIGLLTSLVQELEEEHVRREEELKTTEKLARLEPLAESELAMHLMRERT
SETGESDLLELLDLPADPSGYALVLSFEEEEPHRVSGGPGLWGSGGENGDTASRKHMYDAVKSWCRTRYTCVVSPLMGRH
LAVFVIRPGETEESEGSGRSSHSVPRGTSAEEGARVGGETAAAAATARAAESAGEADSGRDGGNGAGSSRYETAAANAVL
DDSGPDGAAVKARHGRPIPPAPSAEAAEPAAAAGKLHGYVSGRLGRRVTVGAGRAHRGLTGLRRSYAEACAAAINADESA
FVRVYDGEPAGDGPPVFAGNAAAGHGVPGELELAAQRLLSEREERTTSVLKEAKDWIDGHFVEEITLEQAAERVHLSPHY
FSKMFKRHMGATFIDYVTRLRIERAKELMRNEELSLKEVCFEVGYKDPNYFSRVFRKVTGVSPTEYRGR
>lcl|NZ_CP026520.1_prot_WP_042229869.1_1029 [locus_tag=PC41400_RS05450] [protein=pentapeptide repeat-containing protein] [protein_id=WP_042229869.1] [location=1209984..1210592] [gbkey=CDS]
MYSYIEETYTGTDFSGRDLGEGELRNCTFERCRFIGTRLEEAWTESCQFIDCDFSGAQFNASSHKRSAFTNCRFRGANFF
VAKFEECKMIGSDFNEASLDGITIAGGDWSYTNLRFANLSKQAMRGIRLIEADLYSCNLEKADLRDADLTRARMGKSRMQ
GTDLRGAVMDGVDFMSLDIKGARMDAHQGAAFLRAYGARVDD
>lcl|NZ_CP026520.1_prot_WP_042229868.1_1030 [locus_tag=PC41400_RS05455] [protein=ABC transporter substrate-binding protein] [protein_id=WP_042229868.1] [location=1211071..1212402] [gbkey=CDS]
MKRKMATVLSVMTIGTTLIAGCGSGESGGNGAQTTPGASSPAPSAAAGGSKKVEIFSWWTGAGEEAGLKGLIDLFKTKNK
DIELVNAAVAGGAGTNAKAVLASRMQGGDPPGTFQVHGGAELNTGWVAAGKMEALNDLYDKEGWKDKFPKDLIDLVSKDG
SIYSVPVNIHRGNVLFYNKKVFDANGLKAPKTFDEFFQAADELKAKGITPLALGDKEPWTATMIFENVLLGTLGPDDYMK
LWTGEISFEDAKVKQSLETFKKMLTYTNKDHAARNWQDAAQLVAKGDAAMNIMGDWAKGYFTTDLKLKANEDFGWAATPN
TENSFMVITDTFGLPKGVKDPEAVKEFLKVLGSVEGQDVFNPLKGSIPARTDADTGKYDAYGKQTIEEFKKSKLTPSMAH
GSAAPEGFVTQANQAMTIFVTKGDVEQTAKMLQQAAAASGMKK
>lcl|NZ_CP026520.1_prot_WP_241688882.1_1031 [locus_tag=PC41400_RS05460] [protein=sugar ABC transporter permease] [protein_id=WP_241688882.1] [location=1212546..1213439] [gbkey=CDS]
MPILMVLPSFAAILVFVYGFIAWTGYVSLSNWNTLVKDLSFAGLSNFKFLFADFRFQSDLRNTLMFTVLFIAATMAIGIF
LAMMVDAKIKAEALFRNLFIFPMALSFVVTGVVWQWILNPSTGVNLILKSFGMMDPPKWYTSTEIVPGIAWGLIDFGLPL
ALIAVVIAAVWQMSGFTMAMYLAGLRAIPDELREAARMDGAAERQVFTRIILPQLRPVTVSVVIILGHISLKIFDLIFAM
TGPGAMFLTDVPGVYMFETTFRGNHYAQGAAIAIIMLLFVSLLIVPYLFSSLKKEEG
>lcl|NZ_CP026520.1_prot_WP_042229865.1_1032 [locus_tag=PC41400_RS05465] [protein=carbohydrate ABC transporter permease] [protein_id=WP_042229865.1] [location=1213439..1214260] [gbkey=CDS]
MRLAQINRSLLYMLLAVMACLYLVPIYVLGATSFKGLDEITLDQMWRLPSHLDFSGYKTAFDKLAPNFLNSLYLAVPATL
LSALFGSLNGYVLSKWQFRGSNTLFALMLFGMFIPYQSILIPLIQFLQKIHLYNSIPGLILVHVVYGLPICTLIFRNFYA
GVPDEIMEAAKIDGTGFFGVYRYIIMPLSITGFVVVGIWQFTNIWNEFLFAVTLTSQKQQPVMVALQNLSGSQIVQWNVQ
MAGAILAALPTLLVYIFLGKYFVRGLLAGSIKG
>lcl|NZ_CP026520.1_prot_WP_042229863.1_1033 [locus_tag=PC41400_RS05470] [protein=LCP family protein] [protein_id=WP_042229863.1] [location=1214530..1215489] [gbkey=CDS]
MKKKTKWKIAALSVLVIGVSASGWAYWSLDPSRHFENQDIPVLAQPQTAEAAAPGPVQEPPKPDMKSFNILILGVDSRSG
EPSRSDVMMAVHVVPAERKVNILSLPRDTRVPVKGVGLTKLNHAHIVGESKGGNQQGTLVATQTVSDFLGVPINYYAKTN
FEGFKRIVDKVGGVDLTLEKDLHLKFQKTTIPKGQQHFDGDTALKVVRERYGAEGGEFGRQEEQSQVLRSIAIELLKPDK
IPQLATLLPSLRKDILDTNFSDSDLISLAWLFKGVGQDNFRYTQLPGHSGREEDPLVGSNLYYWIPDSESVKKIKEDLF
>lcl|NZ_CP026520.1_prot_WP_042229860.1_1034 [locus_tag=PC41400_RS05475] [protein=tyrosine-type recombinase/integrase] [protein_id=WP_042229860.1] [location=1215689..1216207] [gbkey=CDS]
MQVAQPIRDPEKIIELQGVLKNHSMRDWLLFTIGINSGLHLNDILRLRVKDVKDRNEISITEEKTGKSKSFWIDDALKSY
IGEYVQYMGDEDYMFPSQKTGKPIKRIRVYRILNEAAKRVGLQDIGTHTLRKTFGYHYYQRTKNISVLRDLFNHSAPSVT
FKYIGIDKRNSG
>lcl|NZ_CP026520.1_prot_WP_009672225.1_1035 [locus_tag=PC41400_RS05480] [protein=helix-turn-helix transcriptional regulator] [protein_id=WP_009672225.1] [location=complement(1216576..1216950)] [gbkey=CDS]
MNKHGDRIAQLREKHALTQEELANKLGISRAALSHYETSRREPDFETMNKIATFFKVSADYLLGRTTNPHAVLDEDVRDF
VNSLELSDDKILEKFALSIDGRKLTAEEAKRFIAFVRAERSMSE
>lcl|NZ_CP026520.1_prot_WP_042229858.1_1036 [locus_tag=PC41400_RS05485] [protein=capsular biosynthesis protein] [protein_id=WP_042229858.1] [location=1217427..1218218] [gbkey=CDS]
MYTDLHCHILPGLDDGAPDLEAALTMAEHAAASGISRVVATPHHGGAYENEAPAVREAVELLQQALDRRGIALTVLPGHE
IRVYDRLLDDLLDGRLCTLGGSRYLLLELPGGRVPGGFAELLHELGVRGIVPILAHPERSSEIQRRPELLREWMDGGLLL
QVTAGALTGDYGRRMQAFSLSLCRANAVHFLASDAHDVVVRKWHLADAYALAARKVGPEQVAAYRANADYVARDEVIVPA
LPARNMSLSRALWPWPFNIRVKG
>lcl|NZ_CP026520.1_prot_WP_255246838.1_1037 [locus_tag=PC41400_RS30550] [protein=hypothetical protein] [protein_id=WP_255246838.1] [location=1218364..1218486] [gbkey=CDS]
MKKAYIAPTVLKHQQIVFETKPSKPHPCDNGGHTGSPNCK
>lcl|NZ_CP026520.1_prot_WP_053228707.1_1038 [locus_tag=PC41400_RS05490] [protein=nucleotidyltransferase family protein] [protein_id=WP_053228707.1] [location=1218483..1219607] [gbkey=CDS]
MTQPLIRFMNDLYVVHGAFSFTEEEYGQIAEDLDLFSITAQVYHMLKEKGRLGELPLFLQDHLKRKAEATLFQNLLLKTE
TDRLLRLFEEAGLQVLLMKGIYFSERYYGGFGVRGTTDIDLLIQPENLSAAAAIVETLGFERFDESDASHHVVYTREGAA
GRLPITVEIHWGLVEFRTADIDYDLFWSQSLPLQNYVHIRRMNDSDTFYSICLHGSQHRMNSIRYFLDVVHLLQNADVES
VKKGVVQRAAREYTVKRMKTVLAAVDYLFPGLPVIVPVRPSIPPFWSYSLIRNVQQGHKGSRNALYILLYHLTQVYDRSA
HTGAWFCNFLFPYVHPALLGLDGVRSASRWKHFKSVYAERWKRLTRHVKMKKYV
>lcl|NZ_CP026520.1_prot_WP_042229856.1_1039 [locus_tag=PC41400_RS05495] [protein=PqqD family protein] [protein_id=WP_042229856.1] [location=1219620..1219937] [gbkey=CDS]
MEKKFRRHQDTEVMLLDGEYVIMQMNRQSITRVNEVGGMIWSLLSETRTVHELVLCLQDHYETTYQTAENDIHLFLQQLE
EYDLVEIMEVVEKESAPLPYRSAVT
>lcl|NZ_CP026520.1_prot_WP_042229855.1_1040 [locus_tag=PC41400_RS05500] [protein=Wzz/FepE/Etk N-terminal domain-containing protein] [protein_id=WP_042229855.1] [location=1220072..1220830] [gbkey=CDS]
MELDLKEYLHIIRKRIWLIATIVVLSCLVTGLISYFYIKPQYAASTKLIVNSSYQTEGISRIDYSEVSASIMLVNTYKEI
IRTPAILDKVAAQLPELQLTADEIISMVKVSAVNETQVLTITVTDYSHATAVKLVNAISNVFKTDIPQIMKVDNVTLLNQ
AKQEEAPFPVKPNPKMNIVIAFILSALIGVGLAFLLEYLDDTLKSEKEIEQYLGLPTLAVISRIKASDAPSNVPSKPEQK
VVGESVYAGVNQ
>lcl|NZ_CP026520.1_prot_WP_042229854.1_1041 [locus_tag=PC41400_RS05505] [protein=CpsD/CapB family tyrosine-protein kinase] [protein_id=WP_042229854.1] [location=1220811..1221497] [gbkey=CDS]
MPALTNNRLITHVNPRSNIAEAYRTLRTNIQFSSIDQEIRSVLITSAQPEEGKSTTAANLAVAYAQENKKVLLIDADLRK
PTLHKVFNVSNRSGLTHVLVNQDRFEDALTDTGIPNLTLLPSGPVPPNPAELLGSNRMLTLLEEVGDWYDMVIIDSPPLL
AVTDSQILAAKADGVVLVLNHGKVKRDRAQKALSQLQYVKARVLGVVLNNKAYSKKESSYYYYYGEKA
>lcl|NZ_CP026520.1_prot_WP_042229853.1_1042 [locus_tag=PC41400_RS05510] [protein=sugar transferase] [protein_id=WP_042229853.1] [location=1221825..1222493] [gbkey=CDS]
MYNDQNGLLREPAVSVEVSIREKKYFFGFYSFYTYAKRISDFLLALIGIVLSLPIMLVVALAIRLESPGSVFFMQERVGK
HGRLFKIIKFRSMTMDAEKNGAQWAQKNDSRVTRVGKFIRKTRLDEIPQFINVLKGDMSLIGPRPERPVFTEQFNEEIPG
FINRLEVKPGITGWAQVNGGYEITPKQKLELDLYYIKKSSFRLDMKILLKTVGVVFTGDGAR
>lcl|NZ_CP026520.1_prot_WP_042229852.1_1043 [gene=wzy] [locus_tag=PC41400_RS05515] [protein=O-antigen polysaccharide polymerase Wzy] [protein_id=WP_042229852.1] [location=1222490..1223872] [gbkey=CDS]
MNAWLLKPLGARWKTALLTLFVIILVSICSILLFAHILLTQITVFCLVNIAFIRLYLKLNPALNAFTSVTVLFSFFFLIY
GIALPMDTLFQNINLIQSDEVIGTFKAYLAANLCFVAGVLVSQFLKVRGKDKLVSDTPIDLLKIQAAAIGLLLIGLTLMV
YDQFRLGGFDVIGVENRMNNFVAQRQSTNLAIPWKIMITAGFFGWSISVCSKKQLYLLYFTLISVCLFYFFGLGSRSLIL
MTILPTFAWLVNQRKIKLTSMRKVILVGIVLAMMSPIFNNVRQSIIENKSMDSPKQSWSFSEGETGTAFEITNDILTTSI
YPNADPSYLTAVLYFLPSSIYKFFIDHEKPLNMGDWYVSYFHPYLYQHGGGMGFSPVAQAWMNGEWWGIFGVFIILGFIL
CSLDRTLHIHYLVLPFAIWFQRSSFHSVMTDFIFILIFYSFIRFVGTLGQVSKKPKYKIQ
>lcl|NZ_CP026520.1_prot_WP_042229850.1_1044 [locus_tag=PC41400_RS05520] [protein=glycosyltransferase family 4 protein] [protein_id=WP_042229850.1] [location=1223896..1225083] [gbkey=CDS]
METDRKRIAFVATVFGHLEAFHIPYMTLLKNKGYEVHAFGAVDHGYPGISNRFKSFPISFNKNPFHKDNWEAYMDLVSYF
KENEYSMVHFHTPIASIVGRLAARKAKIPSVVYTAHGFHFHGGAPLLNWLLYYPAERIMARYTDYLITINKEDYLRAKRF
KVRKEIIYLPGIGVDTSAFDSKISQEDSQSLRKSLGMQNGDFIVTCIAELNSNKNQEQLIRAVHILVSRYPHLKCLFIGQ
GNKAEYYKDQVATLQLNDHIQFLGFRRDIPELLAISQVVALVSKREGLPKALLEGMSAGKALFGTDIRGISDLITDGKNG
FLVEVSNIPQSVEVLESMLQDPQRIDEMGLQSKCLVSKYDLSVVISEMNDIYEKALSKNVIGYITANGEPAELSK
>lcl|NZ_CP026520.1_prot_WP_053228706.1_1045 [locus_tag=PC41400_RS05525] [protein=oligosaccharide flippase family protein] [protein_id=WP_053228706.1] [location=1225112..1226407] [gbkey=CDS]
MDSKSDIKAQGPVKSLSLKMNFSWTFAGNLFYAICQWAMLSVMAKLGTAEMVGQFSLGLAITAPVFMFTNLQLRGILASD
RKDEYTYGQYFGLRLTATLIAFLTIVGISFLGPYNWETAVVIVLIGLAKSAEAMSEIAFGLMQKHERMDKISLSLIAKGI
ISLLLLIILLWLTQNIILGMVSFALSAILLFLTYDLTNSRNYDSTKIILNLHELKKLIKTSLPMGLVMLLISLNANIPRY
FIEHGMGTDKLGYFSAISYMMMAGTTVVSALGQSAVPRFAGYYAEAKWGMFRSLMIKLVMIGTILGVSGVFIVLVFGKEM
LTIIYKPEYASYQSVFILIMIGAGISYISSFLGYGLTSARLFSVQPYIFLVVCLTTLLASYWLIPLWGLEGAAVAVIVGN
VINLAGCMTANSFILGRLKGSQEVRTVNEAN
>lcl|NZ_CP026520.1_prot_WP_042229849.1_1046 [locus_tag=PC41400_RS05530] [protein=glycosyltransferase family 1 protein] [protein_id=WP_042229849.1] [location=1226394..1227518] [gbkey=CDS]
MKPIRILHVVRIMNRGGAETLIMNLYRQIDRNQIQFDFAVSSDKPGDYDNEIRELGGRIIIHPHPTTEGIIAFQKSFLNT
LRSHGPYQAVHSHVYTFSGFVLKIAKKENIPVRVAHCHTTQDGHGHNPMRAMYRWYARKMILTYSNHMIGCSKAASEALF
GKNCFSDPRVQVLANGIDLEPYKALNSQAKSEVRRRLGLPVDSRIIGHIGRFSEVKNHKKIIEIFECVVKARPDTQLVLV
GDGPLKAAVEEDVKNRGLGKKIHFMGVRKEIPEILKAMDVFLMPSLFEGVPVVLIEAQASGLPCIVSNTITKAADLSSSH
FKFLSLEDSAEHWADEVIDFLENDLPEWDMREKAIQQAGYDIRTVVNKLSAVYM
>lcl|NZ_CP026520.1_prot_WP_042229847.1_1047 [locus_tag=PC41400_RS05535] [protein=NAD-dependent epimerase/dehydratase family protein] [protein_id=WP_042229847.1] [location=1227615..1228559] [gbkey=CDS]
MKVLVTGGAGFIASHITDKLIEQGHDVVCVDNLSTGTEENINPQATFYKEDITSDNIREIFIIERPECVIHHAAQIDVQK
SIRYPVEDAKINILGTINILESCKEAGTKKIIYASSAAVYGDPQYLGVDEIHSIVPLSHYGISKYTPEHYIKAYEKLYGL
RYTILRYANVYGTRQISKGEGGVISIFVENLLNNNSPIIFGDGEQTRDFIYVEDIAAANVAALTFGDNFVLNVSTNERTS
INQLVNMLNRVANTELPIVYREARNGDIVHSCLDNTLAKLELKWAPKFTLKEGLQKTVSYYALKRELQPQVQVL
>lcl|NZ_CP026520.1_prot_WP_042229846.1_1048 [locus_tag=PC41400_RS05540] [protein=glycosyltransferase] [protein_id=WP_042229846.1] [location=1228580..1229767] [gbkey=CDS]
MKLTFAYDVVIEHLEGTEQYFHNYFDYTVWQRYLSVFDTIEVISRQKEVDSGSRQNLKVSSGPEVTFLKAPSLSDPVSMI
TKRKEVIEIIKKSLSTTDAMIVRLPSEIGPLAVEVAKSIGKPYAVEVVGHAWDALWNYGNWKGKVYAPIMTWKMKKTVKN
APFALYVTREFLQKHYPTKGHSINCSNVELPSLPKSILEDRKVVMDGNNDTIKIGLIGSMVSKYKGIDIALQGLHLIKDE
ISFEFRVIGEGNPTEWDELSRNLGIRNKVKFYSPVPSGEPIFKWLDDLDLYIQPSFQEGLPRALIEAMSRALPAIGSDVG
GIPELLSPDCVFKAGNAEELASILLRMVRDRQKREKESEKNFKTALEYTNLILNERRTTFWGEFSQFAESFKKVM
>lcl|NZ_CP026520.1_prot_WP_042229845.1_1049 [locus_tag=PC41400_RS05545] [protein=NAD-dependent epimerase] [protein_id=WP_042229845.1] [location=1229769..1230782] [gbkey=CDS]
MILVTGAAGFIGFHLVHKLLDEGQIVVGLDNLNDYYDVQLKKDRLAQLGKRPSFHFVEGDLENADAVNSLFEAHKPAVVI
NLAAQAGVRYSLENPSAYISSNIVGFTNILEACRTYKVKHLIYASSSSVYGLNKKIPFSVEDSVDHPASLYAATKKSNEL
MAHTYSHLFGIPSTGLRFFTVYGPWGRPDMAYFSFTQKILQGKPIAVFNHGDMRRDFTYIDDIVEGIVRLVDHPPQSKTG
ADGEKAAPYSIYNIGNNKPVPLMKFIQVLEDCLGQKAVIDFQPMQPGDVKETFADIDALAKEVGFAPSTSIEVGLKRFVE
WYTEYYKTSEKSVTQTR
>lcl|NZ_CP026520.1_prot_WP_042229844.1_1050 [gene=galU] [locus_tag=PC41400_RS05550] [protein=UTP--glucose-1-phosphate uridylyltransferase GalU] [protein_id=WP_042229844.1] [location=1230830..1231723] [gbkey=CDS]
MTIKKAIIPAAGLGTRFLPATKAQPKEMLPIVDKPTIQYIVEEAIASGIEDILIISGRGKRAIEDHFDKSFELEETLEQK
GKLEDLKQIQAISNLANIHYVRQKEPKGLGHAILCAKSFIGQEPFAVLLGDDIVQAETPCTAQLIEVYEKYKSTVVGVQW
VADKDVNKYGIVDAGKEIVDRHVIPVRSLVEKPQPSEAPSNYAIMGRYILTPEIFNILETQQPGAGGEIQLTDAIRRLNE
NRPTFAYEFEGIRYDVGDKLGFIKATIDFALQRPDLSSDLYRYIREIYEKDLILQGG
>lcl|NZ_CP026520.1_prot_WP_042229843.1_1051 [locus_tag=PC41400_RS05555] [protein=UDP-glucose/GDP-mannose dehydrogenase family protein] [protein_id=WP_042229843.1] [location=1231728..1233062] [gbkey=CDS]
MSTIGVIGTGYVGLVSGTCFSEVGHQVICCDIDEAKIAKLQRAEIPIYEPGLEELVKRNTNLNRLSFSSDIASVIHRSDV
IFIAVGTPMSPSGEADLSYVRAAAAAIGQSIRDYKVIVTKSTVPVGTGRLVQEIIRSNQSDPSIPFDVASNPEFLREGSA
VHDCMNMERAVIGATSDRAAEIIARLHDPFRTTIFKTSLESAEMIKYAANTFLAAKISFINAIANICERVGADVTEVAKG
MGLDSRIGNKFLQAGIGYGGSCFPKDTHALKHIADVSGYDFKLLEAVIETNDKQRMLVIEKLEKAIGNLTDKKIAVLGLA
FKPNTNDVREAPSLTLIPKLLEKGARVQAYDPIAVEEARCYLPDSVVYTQDLYDAVRDCDACVILTEWKEVVTMELDHIR
ELLRKPVIIDGRNCFALERMRQEGFEYYSIGRPEVLAPNAVLSV
>lcl|NZ_CP026520.1_prot_WP_042229841.1_1052 [locus_tag=PC41400_RS05560] [protein=VanZ family protein] [protein_id=WP_042229841.1] [location=1233320..1233859] [gbkey=CDS]
MTHRSRLRFFVLFGLPLLVWMAVIYHFSSEPYSKQNLRPWLKQEFSDKRINQLFSESKIQYNGKEVSIRKSGTWGFVEFF
IRKAAHLFVYGVLAVLIARLLHRGLGQSKLAVYVTVILASALFAGLDEFHQSFVSQRTAATEDVILDTCGALLGTAMYYF
FLKILQKSRKNAPNYPKTS
>lcl|NZ_CP026520.1_prot_WP_042229840.1_1053 [locus_tag=PC41400_RS05565] [protein=hypothetical protein] [protein_id=WP_042229840.1] [location=1233949..1234167] [gbkey=CDS]
MKALDLVYCIHCGSIVSRIQADHIFKTGFYRVSIPLASCKGCAAPNSNTTSCHDLPHHGRLNEYETRLPLMA
>lcl|NZ_CP026520.1_prot_WP_042229966.1_1054 [locus_tag=PC41400_RS05570] [protein=DUF2621 domain-containing protein] [protein_id=WP_042229966.1] [location=complement(1234484..1234897)] [gbkey=CDS]
MWFITFWPLVLIGLMAIGGYFMFRKFLKVLPKADGKSKLDWQNYYVEKSRHMWTDDSKAFLNKLVAPVPGPFRDIAKHSI
AAKIGQIALEEEAAEVTREHCIKGYIIATPKRDYRSLMNFLEREKIDYTPYKHLINS
>lcl|NZ_CP026520.1_prot_WP_042229838.1_1055 [locus_tag=PC41400_RS05575] [protein=deoxyribonuclease IV] [protein_id=WP_042229838.1] [location=1235066..1236193] [gbkey=CDS]
MIKIGSHVSFSDKGLLNAAREAVSYGSGTFMIYTGAPQNTRRKPIETQYIEEGREVMQEHGIGEIVVHAPYIINLGSYKD
NTYELAVQFLQDEIRRTDYLGVKNIVLHPGAFTDKDAEYGLARISEGLNEVLAGVKDTGVHIALETMAGKGTEMGRSFEE
LAQIIDKVEDNGRLTVCLDTCHIHDAGYDVVGDLDGVLDQFDRVVGLDRIGVVHLNDSKNPRGASKDRHAPVGAGWIGFE
AMNRVVHHEKLLGLPMILETPWIGKDPKTQRPMYEAEIALLSGTVLERFGASFVEDVERLASFFEKKEVDSRGFVLQTWE
LLKTDAKAKKADGREPMERLYDMIAEERLFPELSEEQINQRLTAWFAGKDLLKSL
>lcl|NZ_CP026520.1_prot_WP_042229837.1_1056 [gene=purU] [locus_tag=PC41400_RS05580] [protein=formyltetrahydrofolate deformylase] [protein_id=WP_042229837.1] [location=1236276..1237163] [gbkey=CDS]
MAAANTSGAAGKQNRGRMLISCPDRPGIVAAVSKFLSDYGANIVQSDQYTMDPEGGMFFIRIEFDLTDLEQRKPRLEADF
AGVAAEFSMDWSLSLASQRKRLAIFVSKEDHCLLELLWHWRAGDLDADIAMVVSNHPDMAPLVEPFGIPYHHIPVTPDTK
AEVEKRHLELLEGQVDLIILARYMQIIPPSFIKHYQNRIINIHHSFLPAFVGGKPYAQAHGRGVKLIGATAHYVTEELDG
GPIIEQDVQRVSHRDNVDDLKRIGRHIERIVLARAVSWHTEDRILVHQNKTVVFS
>lcl|NZ_CP026520.1_prot_WP_042229836.1_1057 [locus_tag=PC41400_RS05585] [protein=hypothetical protein] [protein_id=WP_042229836.1] [location=complement(1237286..1237747)] [gbkey=CDS]
MSRPFFENGNTLVARGFAFGLVPGAALALFFKAVEFFAGSRVYTLLLNVDFIPGLGGDQPEWFELTLHLAVSVLIGFLYA
ILLRCTEKPWLPGILLGISPIPLFVPLTQLAVSAKTPVLSDVTAMIWWIAGHLLYGVILAVLGTKALRASKKE
>lcl|NZ_CP026520.1_prot_WP_042229835.1_1058 [gene=tkt] [locus_tag=PC41400_RS05590] [protein=transketolase] [protein_id=WP_042229835.1] [location=1237902..1239923] [gbkey=CDS]
MSASNTPIQELAVHTIRTLGIDAIEKAKSGHPGIVMGAAPMAYTLFANQMKHNPANPTWINRDRFVLSAGHGSMLLYAIL
HLTGYDLPMEEIKNFRQWGSLTPGHPEFGHTAGVDATTGPLGQGIGMAVGMAMAEAHLAGLYNREGYPVIDHYTYAICGD
GDMMEGVSSEAVSLAATLKLGKLIMMYDSNDISLDGDLHVSFTENVQKRFEAYGWQVLRVEEQNDLEALNRAVEEAKADT
TRPTLIEVKTTIGYGSPNKGGKGGHAGPHGSPLGLEEAKLTKEFYKWNYEEDFYVPEEVREHYAAVKEKGIQAEQAWKKQ
FDDYRKAFPELAEQFELGESGKLPEGWDRDLPVYTPEDKALATRAASGNALNAIAKNVPFFFGGSADLASSNNTEIKNAP
IFNAENYAGRNIWFGVREFGMGAALNGISLHGGLRVYGGTFFVFSDYLRPSIRLAALMKQPVIYVFTHDSIGVGEDGPTH
EPVEQLAALRVIPGMTVLRPADGNETSEAWRYVMEHNDGPYALILTRQNLPQLEGSAKYARENLSKGAYVVSDAPNGKPD
ALILATGSEVQHGVAAQKLLAEEGIQVRVVSMPSWELFEKQSQEYKDSVLLPDVKKRLGVEMAYPLGWERYVGDEGSILG
ISTFGASAPGDLLIKEFGFTAENVAKRVKELLK
>lcl|NZ_CP026520.1_prot_WP_009672106.1_1059 [locus_tag=PC41400_RS05595] [protein=pyrimidine/purine nucleoside phosphorylase] [protein_id=WP_009672106.1] [location=1240003..1240320] [gbkey=CDS]
MSNFEGVTVVKKANVYFEGKVTSRAVLFADGTKKTLGIMLPGEYEFGTDSKEIMEILAGDLSVLLPGESDWTDIQGEGEF
TVPANAKFKLKVRTVTDYCCSYIAE
>lcl|NZ_CP026520.1_prot_WP_042229834.1_1060 [locus_tag=PC41400_RS05600] [protein=NAD(P)-dependent oxidoreductase] [protein_id=WP_042229834.1] [location=complement(1240904..1241803)] [gbkey=CDS]
MKQIGFIGLGTMGAPMALNLLRKGYAVTVYNRSADKAAELVQAGASLAATPAEAVSGAEVLFTMLSNDSAIESVFYGENG
VLEKAGEGLTVIDSSTVSPALSQRLHADLGKRGAYFLDAPVTGSKPAAEAATLVFMVGGDRPVFDAQHALLSDLGSKVLY
MGPSGSGSYTKLAHNTIVGINAIGLMEGLSIATKAGLAPEQFLQIVQAGSGNSKQAELKGPKILERDFSNQFSLKLMLKD
LLLASDLTGKFQLPAPMLHAATTVFQMGLSKGLGEEDLAAAIQCYEEWMNERVEKKGGQ
>lcl|NZ_CP026520.1_prot_WP_042229833.1_1061 [locus_tag=PC41400_RS05605] [protein=glucose-6-phosphate isomerase] [protein_id=WP_042229833.1] [location=1242164..1243519] [gbkey=CDS]
MTQKLQFDYSKALDFVAQHEIDYLAGPIRTAHDQLHNGTGAGSDYLGWIDLPVAYDKEEFGRIQKAAKQIQSDSEALVVI
GIGGSYLGARAAIEMLSHSFYNLLDKDKRKTPAVYFVGQNISSTYVTHLLQLLEGKDFSVNVISKSGTTTEPAIAFRIFR
ELLEKKYGKEAARKRIYATTDREKGALKQLANGEGYESFVIPDDVGGRYSVLTAVGLLPIAVAGIDIEEMMKGAADAREA
FSNPELSENASYQYAAVRNALYRKGKTTEILVNYEPSLHYVSEWWKQLYGESEGKDNKGIFPAAVDFTTDLHSMGQFIQE
GTRNLFETVLFVDQSAEEIVIGTDEENLDGLNFLSGKTMDFVNKKAFQGTMLAHSDGGVPNLIVTLPDTSAHTFGYMVYF
FEKACGISGYLLGVNPFDQPGVEAYKKNMFALLGKPGYEEDKAKLEARLND
>lcl|NZ_CP026520.1_prot_WP_042229832.1_1062 [locus_tag=PC41400_RS05610] [protein=YigZ family protein] [protein_id=WP_042229832.1] [location=1243716..1244354] [gbkey=CDS]
MIASYRTVQKHGADEIIIKRSRFIGQALPVSSEEEALAFIAGVKKEHSMATHNCSAYMIGERNEIQKQSDDGEPSGTAGK
PILEVIRNMDLKNVAVVVTRYFGGIMLGAGGLIRAYTDGAVAGIQAALPVYKVSHQELLAEIDYTWLGKVENELRNRNMR
MGDTEFTDKVILRCYPLSAEADSFADWFTDLTQGQGNIIRGEHTYIEHSELP
>lcl|NZ_CP026520.1_prot_WP_042229830.1_1063 [locus_tag=PC41400_RS05615] [protein=TetR/AcrR family transcriptional regulator] [protein_id=WP_042229830.1] [location=1244389..1245030] [gbkey=CDS]
MARKAVAQELSRERILEEARELFVTRGYRALTMRSIAKAMGYSHGALYYHFREKAELFNALILEDFSILLQRQRQIIVES
AVLGVPLLQKLMMEFIRFGLENPNHYEIMFMIRDEELRQYSRTEQAKCFDLFATVVRQVISGMPDKEDQEYRLPWNLFMS
MHGFISHSIHYGQTYELVEQLAEEHVQLLCSNMVSTAARSESSKVSLVNHESA
>lcl|NZ_CP026520.1_prot_WP_042229829.1_1064 [locus_tag=PC41400_RS05620] [protein=secondary thiamine-phosphate synthase enzyme YjbQ] [protein_id=WP_042229829.1] [location=complement(1245011..1245412)] [gbkey=CDS]
MLHTYDLVTRKRDEMVEITDIVARLVQDMSLEDGTAIVYCPHTTAGITINENADPDVKRDVLMRLDEVFPWEHPRDRHAE
GNTAAHLKAMTVGASQTVFIRGGRLQLGRWQGIYFCEFDGPRSRTFTVKLIRD
>lcl|NZ_CP026520.1_prot_WP_042229828.1_1065 [locus_tag=PC41400_RS05625] [protein=DUF1128 domain-containing protein] [protein_id=WP_042229828.1] [location=1245617..1245841] [gbkey=CDS]
MDLAVWSEQNVEYMFDEIKTKLRMATGGSIKASNFSQEQYEDLKDLYDLVMSKPNFSISEIDAITTELGKLRKA
>lcl|NZ_CP026520.1_prot_WP_042229827.1_1066 [locus_tag=PC41400_RS05630] [protein=hypothetical protein] [protein_id=WP_042229827.1] [location=1246078..1246368] [gbkey=CDS]
MSSKLRNDASKLVGQNIVAVRKDGSMVRGKLVRIQGNELILEPQEGTAQTKAFVPLVLFDLLAIGTSPFVNGFGYGGGFG
GGFGYGGGFGGPGFFW
>lcl|NZ_CP026520.1_prot_WP_042229826.1_1067 [locus_tag=PC41400_RS05635] [protein=hypothetical protein] [protein_id=WP_042229826.1] [location=1246531..1247217] [gbkey=CDS]
MSRGKNEFDYEKEWERKFGFRSGEIPSFSSIFGGGETLFTELSKRISEDPENTAWLDQWLDKLLRQAMPGYRIPEWMGTD
WENEPAGGPAGQNAQGSTMPRGGEKRRASEDHSAAAKPPAAIPIPGLKPVKGLAAGSKKRQTAKVKAAESFAYSITEDGS
RLVVIVQQPSGIRPGEIELAADRRSIRLSGSRVGGKQTIRLFRPIRPGSCRMTSEDDSLVIRADLARR
>lcl|NZ_CP026520.1_prot_WP_042229825.1_1068 [locus_tag=PC41400_RS05640] [protein=alpha/beta-type small acid-soluble spore protein] [protein_id=WP_042229825.1] [location=complement(1247304..1247594)] [gbkey=CDS]
MARRRSRRAAVPGSERSLDKLKGQVMRDAGYNVSADQPNLVKYEVARTLGVPLEPGYNGQLRTEDAGKIGGEIGGRMVRE
LVRMAQKQLAERGGPK
>lcl|NZ_CP026520.1_prot_WP_042229824.1_1069 [locus_tag=PC41400_RS05645] [protein=glycosyltransferase family 2 protein] [protein_id=WP_042229824.1] [location=complement(1247882..1248898)] [gbkey=CDS]
MSSPILTIVVPCYNEEEVLPETVSRLSAVLDRLIQEKLVARESVMMLVDDGSKDRTWDLIDQYSRSHDYVTGVKLARNAG
HQNALLAGLMTARHTSDIVISIDADLQDDTDAIREFVRKYMEGAEIVYGIRSDRTTDTWFKKTTAESFYKLMLRLGVQIK
NNHADYRLMSKRALDYLSRYKEVNVFLRGIVPLLGLRTAEVMYERKERFAGESKYPLRKMLSFALDGITSFSVKPIRMVT
VTGFLFSIISLLAGAYAIISKLWGTTASGWTSLILSVWFIGGVQLLALGLIGEYIGKIYREVKHRPLYLIEKSIPTQQDY
EGPADFDRIPVGAERAGR
>lcl|NZ_CP026520.1_prot_WP_042229822.1_1070 [locus_tag=PC41400_RS05650] [protein=catalase] [protein_id=WP_042229822.1] [location=1249155..1250654] [gbkey=CDS]
MNPHRPYLTTNQGLPVSDNQNSKTAGTHGPTLLEDYHLLEKLAHFDRERIPERVVHARGAGAFGVFRPYASMSEYTRAAF
LQDPSIETKVLVRFSTVIHPSGSPETLRDPRGFAVKFYTSEGNYDLVGNNLPVFFIRDALKFPDMVHSLKPAPHTNVQTP
NHLWDFMSLSPESTHMLTWVFSDLGTPANYREMDGFGVHAFKWVNAEGKVVYVKYTWKSLQGVRSLSRKEAQEQGGRDHN
HATRDLYDAIEAGSYPEWELNVQMLEPERLDSFSFDPLDPTKEWPEDLVPFVKVGRMTLNENPKNFFAEVEQSAFAPSVL
VPGIEPSEDKLLQGRLFSYPDTQRHRLGPNYMQIPVNCPYAPVRNYQRDGQMAVRQDHSPINYEPNSYDHLPKEAGSAYA
ESEKPLTGSTTRRKIEKTDDFTQAGERYRRFTPEERANLIDNLVNDLKQTDMQIQLRAVCNFFRADVEYGTRLAEGLGID
LGGFIPQGAGSGSGQMPPR
>lcl|NZ_CP026520.1_prot_WP_042229821.1_1071 [locus_tag=PC41400_RS05655] [protein=ATP-binding protein] [protein_id=WP_042229821.1] [location=complement(1251077..1252600)] [gbkey=CDS]
MDNLGDYFKSFFANISTLLTFTYLGSLAYKYILYRVPVNWMKVLYILLSIAAGWLSMLYGARLTPSVIFDLRFVPILAAP
LFLSRPSHIMLIGIGIGLTRLSFGLDHAAWAGFGNMVILGIVTALTSELLKRTSLRQTVRILLLILIVNLFNVANIAILG
VYPVATYLIELAPLSGTAGLFFSFFFTFILHDFKADFLRREALNKTNQELAEQFRLAEERTAELQQIHDELEVKNEELVA
ASRYKSEFLANMSHELRTPLNSMLILSQFLAENQNGTLSEEEIEFAGIIHSSGQDLLTLINDILDLSKIEAGHMSITPGE
VNVSEVVEAMTRTFEPVAAQKGLYFHIRSTGVPDIIHTDEQRLQQILKNLLSNAFKFTKRGGVTLTFYIEMDAKGLQHPA
AAIRDTGIGIRADKREQIFEAFYQADGTTSRKFGGTGLGLSISRQFARLLGGEIRLTSEEGTGSTFTLLLPSSSLSGQER
ELRQLVDDAKLPRLTPLSSPLPEESNR
>lcl|NZ_CP026520.1_prot_WP_042229820.1_1072 [gene=queF] [locus_tag=PC41400_RS05660] [protein=preQ(1) synthase] [protein_id=WP_042229820.1] [location=complement(1252821..1253315)] [gbkey=CDS]
MAGRNQEELDLTLLGNQQTQYPFNYAPQVLESFDNKHPYRDYFVKFNCPEFTSLCPITGQPDFATIYISYMPDAKMVESK
SLKLYLFSFRNHGDFHEDCMNIIMNDLIKLMDPKYIEVWGKFTPRGGISIDPYCNYGRPGTKYEQMAEHRLMNHDLYPEK
VDNR
>lcl|NZ_CP026520.1_prot_WP_042229818.1_1073 [gene=queE] [locus_tag=PC41400_RS05665] [protein=7-carboxy-7-deazaguanine synthase QueE] [protein_id=WP_042229818.1] [location=complement(1253569..1254396)] [gbkey=CDS]
MSASQTNAAVPSPPRTLPAAGKTAAAAVSPSGTAAAGEATIPVLEIFGPTVQGEGMVIGRKTMFVRTAGCDYRCSWCDSA
FTWDGSGAADIRPLTAEAVWEELVRTGGSRFSHVTLSGGNPALLRSLGPLVKLLRAQGIRTAVETQGSRWQDWLTEIDEV
TLSPKPPSSGMETDFAKLDDIVRRLNPDGVSLKVVVFDERDFAFAREVHRRYPAVPFYLQAGNEDVRLADDDLLRGQLLR
KYEWLVDLTMASEDLNDVRVLPQLHALLWGNKRGV
>lcl|NZ_CP026520.1_prot_WP_042229816.1_1074 [gene=queD] [locus_tag=PC41400_RS05670] [protein=6-carboxytetrahydropterin synthase QueD] [protein_id=WP_042229816.1] [location=complement(1254393..1254881)] [gbkey=CDS]
MKAPGEFRIVENLQQYGQDIRHEQLRYHTKRVLVSKEFTFDAAHHLHAYEGKCKNLHGHTYKAVYGISGYVNEIGLTLDF
GEIKDIWKARIEPYLDHRYLNETLPPMNTTAENMVVWLYEQMETALQSDEMQAKARGGRVEFVRLYETPTSYAEARREWM
SE
>lcl|NZ_CP026520.1_prot_WP_042229815.1_1075 [gene=queC] [locus_tag=PC41400_RS05675] [protein=7-cyano-7-deazaguanine synthase QueC] [protein_id=WP_042229815.1] [location=complement(1254878..1255543)] [gbkey=CDS]
MKQEKAVVVFSGGQDSTTCLFWTLKHFGEVETLTFDYGQRHKLEIECAARIARELGVKNTVLDMSLLNQLAPNALTRGDI
EIEEKEGELPSTFVEGRNLLFLSFAGVYAKVAGSRHLVTGVCETDFSGYPDCRDAFVKSLNVTLNLSMDYPFVIHTPLMW
LNKAETWKLSDELGAFDFVRTKTLTCYNGIVGDGCGECPACKLRSAGLQNYLLEREGADAR
>lcl|NZ_CP026520.1_prot_WP_042229964.1_1076 [locus_tag=PC41400_RS05680] [protein=SDR family oxidoreductase] [protein_id=WP_042229964.1] [location=1256028..1256849] [gbkey=CDS]
MHGNRPDSANKAVERAKEGRAPVAIVTGVSRRIGIGAAICRALAAKGMDVMFTSWTAFDKGMPWGADEEGPDAILRELAE
SGIRCERLEIDQSDPEAAMRIVDETEKRLGFPSVLVNNAAYSVNDGYEKLDAATLDAHYAVNMRGTALLSVEFGRRFNRG
SGGRIINLTSGQSLGPMRGEIAYAATKAAVEAMTVTLAAELAPKGITVNAVNPGPTDTGWMTEELRNGLADQSPFGRIGK
PEDAARLISFLAGEEAEWITGQILHSEGGFRLS
>lcl|NZ_CP026520.1_prot_WP_174446813.1_1077 [locus_tag=PC41400_RS05685] [protein=cupin domain-containing protein] [protein_id=WP_174446813.1] [location=complement(1257004..1257351)] [gbkey=CDS]
MTTQNLLELTRDIQERHVNFAVADVNSHVLRIAVMTGNYRWHFHPDSDELFMVLEGELFIDIEDGTTASLKANDTLTIPA
GTIHRTRADVRTVNLCFEGKEAETIFVDEPAGQAK
>lcl|NZ_CP026520.1_prot_WP_042229812.1_1078 [locus_tag=PC41400_RS05690] [protein=DUF1992 domain-containing protein] [protein_id=WP_042229812.1] [location=complement(1257354..1257746)] [gbkey=CDS]
MSFLDSLVEQRIQEAIRNGDFDNLPGKGKPLVLEDWSAVPEELRLSYKIMKNSGFLPEEMQLRKDMITLEELLACCHDEE
ERGKLKKTLNAKKLRFNMLMEERSMNLSPALARYRDRIETKLSAEPDKPK
>lcl|NZ_CP026520.1_prot_WP_042229811.1_1079 [locus_tag=PC41400_RS05695] [protein=hypothetical protein] [protein_id=WP_042229811.1] [location=complement(1257872..1258099)] [gbkey=CDS]
MSENQQTPQSAETQPVEGAEPRKVSLAEAMKQKMAQKKAAQAQANSQQKHQVTGNQMMKSQQTKKQNNQRRRMGV
>lcl|NZ_CP026520.1_prot_WP_009672054.1_1080 [locus_tag=PC41400_RS05700] [protein=YwbE family protein] [protein_id=WP_009672054.1] [location=complement(1258166..1258366)] [gbkey=CDS]
MNGQSRKNIHAGLTVDIVLKQDQRTGKRTRGVVKDILTNSPNHPHGIKVRLTDGQVGRVQEIVQEP
>lcl|NZ_CP026520.1_prot_WP_042229809.1_1081 [locus_tag=PC41400_RS05705] [protein=RluA family pseudouridine synthase] [protein_id=WP_042229809.1] [location=complement(1258379..1259332)] [gbkey=CDS]
MRKQRPPYNPKTKTARPATPKTDKEPAQFEIGQPAELMKFLLEKLASKGRNTVKGILSRGQVSVNGKKTTRHDHPLQPGD
RVTVQWSVVTEEEKPIGLHILHEDEDLLVINKEAGLLSISGGDEKELTAYWQLTEHVRRKDPKGRIFVVHRLDRDTSGVM
LFAKNEKTQQALQNAWQEAVQERSYVALVERKVKKPEDTIVSWLKESKTLLMYSSPVDNGGQKAVTHYKVLQASSDYSLL
EVQLETGRKNQIRVHMQDIGHPVVGDKKYGSVKNTIGRLGLHARVIAFEHPSTGQPMRFETQIPGKFLRVFAGKPTS
>lcl|NZ_CP026520.1_prot_WP_042229807.1_1082 [locus_tag=PC41400_RS05710] [protein=ABC transporter ATP-binding protein] [protein_id=WP_042229807.1] [location=1259540..1260511] [gbkey=CDS]
MPQTDGLSDEILIDVDGLVKTYGSFTAVNGVRFQVRKGEVFGLLGPNGAGKTTTIEMMEGLRKPDGGTASIAGYDTIRDL
RKVKEVIGVQLQSTSLFDLLKVREIVEMYASFYPKSVPVGPLLADMTLEEKAGDRVKHLSGGQKQRLAIAIALVNDPQVL
FLDEPTTGLDPQARRTLWDIILRLKEKGKTIVLSTHYMDEAHVLCDRICLIDKGQVVALDTPANLVRSLQSENAIEFRLP
QADLASPRERTELTDTLRKVQGVKQVELARDTLVLYTDHLQQSLTGLIREAELRGWKLTDLLTRTATLEDVFIHMTGRSL
REE
>lcl|NZ_CP026520.1_prot_WP_042229805.1_1083 [locus_tag=PC41400_RS05715] [protein=ABC transporter permease] [protein_id=WP_042229805.1] [location=1260508..1261608] [gbkey=CDS]
MKAYIQLTLAQLRIFIRNKQVLLWSFLFPVFFMVMFGTFFGGGSSMSINGVVIDQDRSAQSAQLVGALMKTPAMKLKQEE
HTAPGLQKLKDGGLQMVVIIPAGYGNKLDAAVKGGEASSVTVYYDQTNMMTAELGKTVVAQVADGISKEKTAYKPAITVK
AEGVQALKLSYIDFLVPGIVSMMIMSNNLNGVAGQIASWRERGILRRMQSTTLKSSTFIASQITARLLLNGLQAVIVLLI
ASVIFGTKVNGSWGLLLLFVILGTLAFMSIGFIVASLAKTPESAGPIAGFISFPMLFLGGVFFSADNIPSFLQPVVNLIP
ITHLSTAMRQVMNVGADLTVLWPQALILGGWMVVAFAIASFTFKWE
>lcl|NZ_CP026520.1_prot_WP_042229803.1_1084 [locus_tag=PC41400_RS05720] [protein=stalk domain-containing protein] [protein_id=WP_042229803.1] [location=complement(1261706..1262419)] [gbkey=CDS]
MKSKKPVLYLTGGILIGTLFSQTDVWAKPVIDYLLQAATYPVTVDGQTYISPDLPLLNYEGSTYVPLKSLGDLLGASVTW
NDSLKQAEILKTKPSAAKADLSVATKDYYGAVEKWTFDTLYQPGKETLDEKGRGELLKQLETVYGPNLAARFFDYGYMKE
SDGSYLSRPGDHLPLRKLDRNIIATHEIAGGKTIVTIKGEIEYNLDGKTFTERYKQQYTIDPLTLKIIDIAYLALPI
>lcl|NZ_CP026520.1_prot_WP_042229801.1_1085 [locus_tag=PC41400_RS05725] [protein=hypothetical protein] [protein_id=WP_042229801.1] [location=1262634..1262894] [gbkey=CDS]
MILYQLTQWLQEHTSIRKAIRQEAGDPGELMQRKEQVERQLMEQAERWRNVYADVPALPAADQFPSWSSESGAGWEDGEP
EFDTPR
>lcl|NZ_CP026520.1_prot_WP_042229800.1_1086 [locus_tag=PC41400_RS05730] [protein=radical SAM protein] [protein_id=WP_042229800.1] [location=1262998..1264779] [gbkey=CDS]
MNIVLATLNAKYIHTSLALRYLKAFSEPEFPIDIVEYTIKDPAMNIVSDLYGKNPDVLGFSCYIWNIEETLRVIRMIKKI
RPDLTIVLGGPEVSYDTEEWMRTVPEVDFIVMGEGEETFKHFLQELSTTRKFHFVYGIAYRKGEEIILTPGRPKLDLNAI
PSPHRFEEDLKDLGNRVVYFETSRGCPFSCQFCLSSIEVGVRYFDMERTKSEILYLIANGAKLIKFVDRTFNIKRDYALE
IFQFLIDNHQGCVFQFEITADIMRPEVLDFLAENAPPGIFRFEIGVQSTNDETNNLVKRRQNFAKLTRTVTKVKESGKID
QHLDLIAGLPEENYDSFRKTFNDVFALGPEELQLGFLKMLRGTGLRIDAAKYGYIFMDQAPYEILGNDILPFTDLIRIKR
VEDVLEKYWNAHRTDHTVNYLIASEFPSAFDFFQEFGDYWEERGWQKIGHQLEDLFTRLHAFLTWRGTKGMGVIEGLMKL
DYFLNHKYKPRKIWWDFKLDKAEQSDYLKRAADRPDEVSGDFSELRMGEKDLHKHAVIEVVPFDLRRYLQDGSLDTSAES
LLLAVYRPEAGGRPTVYSMPLGRASAPSVSAGN
>lcl|NZ_CP026520.1_prot_WP_042229797.1_1087 [locus_tag=PC41400_RS05740] [protein=LacI family DNA-binding transcriptional regulator] [protein_id=WP_042229797.1] [location=complement(1265198..1266175)] [gbkey=CDS]
MANIKDIARLAGVSVATVSRVLNDHPYVSEAKRAAVQEVVQRLNYSRNVNAVHLITGKTNMVGVVIPYLNLPYFNSIMEG
IAQEALEANVHLLVSQTNYAADKETELLNMLKIKQIDGLLICSRSLKWEVIEAYSQFGPIIACEYIESDRISSVHIDHYG
SFRLAMQLLLEKGHTRIGYCLGRGHSFNSRNRRRAFSDALSSAGVPVREEWMFYQIYSIEDGAEVVRSILDMEERPTALL
VAGDQVAAGIISEAVKNGLRVPEDLAIVSFDNHPIAQLFDITTIDNRLVEIGRTAFSLLLKRLQDPASPVEHRELPFRLI
ERATV
>lcl|NZ_CP026520.1_prot_WP_042229963.1_1088 [locus_tag=PC41400_RS05745] [protein=Cof-type HAD-IIB family hydrolase] [protein_id=WP_042229963.1] [location=1266412..1267221] [gbkey=CDS]
MERAIVFFDIDGTLLDENKQIPASTRKAVRLLQERGVYTAIATGRVPSMFKDIREELGISSYVSINGQYVLFEGREIYTN
PMDPEKLAELVRAADEQGHPLAYCGHPEIRVSSSHPHVKESYDNLKLPCPETDPEFYKHSAVFQGHLFCDAESEKLYAER
YPYFQFIRWHEHALDVLPAGSSKAIGIQEMLKAVGIPNENCYAFGDGLNDIQMLTSVGTGIAMGNAAPEAKAAADLVTDS
CSEDGILKGLVQVGLLPEEALEELGEHAR
>lcl|NZ_CP026520.1_prot_WP_042229796.1_1089 [locus_tag=PC41400_RS05750] [protein=class I mannose-6-phosphate isomerase] [protein_id=WP_042229796.1] [location=complement(1267347..1268312)] [gbkey=CDS]
MKPYPIHFQPDFKHRVWGGRALEEFGYEIPEGHVGEGWMIADHANGTSTVTNGELKGLGLDEVREKLGAEWFGAKGASAG
NGRFPLLVKLLDCNDDLSVQVHPGNDYENLAQGELGKTEMWYVLDAKPGAKIIYGLKDGITRSEFEAAVQEDRILDTMMS
VEAEKGDSFYIPSGTVHALGSGVLVAEIQQNSDTTYRLYDYNRPGLDGKPRELHIEDSLNVIGFGKAGATFMKTMDAAPN
EWLELAASPYFVVEKGVVTAPWNLSTTPESFTLLIACEGSGSLEWADGSLAMKPGDCFLLPATLGGYTLSGSQTVLRSYL
P
>lcl|NZ_CP026520.1_prot_WP_042229794.1_1090 [locus_tag=PC41400_RS05755] [protein=class I SAM-dependent methyltransferase] [protein_id=WP_042229794.1] [location=complement(1268309..1268914)] [gbkey=CDS]
MGFLSILSCAHKWIEGRVRPGDTVIDATVGNGVDTIFLARLVGSRGRVYGFDIQPQALETARTRLERELDGSGGAQRVSL
LLRSHADMAAAVPGECLGRTAAVMFNLGYLPGADPEVITVPASTLPALEAALLLLHEGGILTVALYPGHDGGASEAQAVE
SWAAGLPQEKWSVLCYRFINSAARSPYLIGIEKKKPRGVLK
>lcl|NZ_CP026520.1_prot_WP_042229792.1_1091 [locus_tag=PC41400_RS05760] [protein=TIGR01212 family radical SAM protein] [protein_id=WP_042229792.1] [location=complement(1268927..1269895)] [gbkey=CDS]
MNAITTEPLLWGDKRFHTWNYEMRRQFGEKVFKVMLDAGFTCPNRDGKIATGGCTFCSARGSGDFAGRRRDDLVTQFNTI
RDRQHQKWPDAKYIGYFQAYTNTYAPVDELREYYEVILQQPGVVGLSIATRPDCLPDDVVEYLAELNERTYLWVEMGLQT
IHESTSTLINRAHDTQCYTEAVAKLRKHNIRTCAHIIYGLPQETDDMMIETGKAVAAMDVQGIKIHLLHLMRKTPMVKQY
EAGLLRFLEQDEYIKLIVDTLELLPPEMIVHRLTGDAPRDLLIGPMWSLKKWEVLNAIDAELKRRDTWQGKLWTPAAPAG
PA
>lcl|NZ_CP026520.1_prot_WP_042229791.1_1092 [locus_tag=PC41400_RS05765] [protein=WGxxGxxG-CTERM domain-containing protein] [protein_id=WP_042229791.1] [location=1270056..1270655] [gbkey=CDS]
MKKLRYATLAVALTSMLTVGAAHASTLGTGSTTGTTNTTVTGTTTPGTTMNGVNPNATAPDQLNRAYNTGRNAVTGTVGA
VTGGVTGAVTGAVNGAATGAATGAATGTTSGVNTYGTGHNVNTGTYGTTGTVGTNGNRMMDRMTTTGTNNRGAYDTSTYR
TYGTTRNNNMNWGWLGLLGLIGLAGLGGRNKRENEIHNR
>lcl|NZ_CP026520.1_prot_WP_042229790.1_1093 [gene=trmB] [locus_tag=PC41400_RS05770] [protein=tRNA (guanosine(46)-N7)-methyltransferase TrmB] [protein_id=WP_042229790.1] [location=1271002..1271733] [gbkey=CDS]
MRLRGRKGIREAIEKQRDLIVLEPALYKGKWAEVFGNGNPIHVELGMGKGQFISEMSRRNPGINFIGVDMYDELIRRASE
KVRIAHGIEESAVDVSLENIRLALFNIERLEEVFDENELERIYLNFSDPWPKKKHARRRLTHAGFVHKYKHALNERGEIH
FKTDSRSLFEFSLNSFADLDLRMRNISLDLHGGFEEGFNPDHVMTEYETKFAKQGMNIHRVEVVLGSRALAEHVERLQEE
QQS
>lcl|NZ_CP026520.1_prot_WP_042229788.1_1094 [locus_tag=PC41400_RS05775] [protein=UDP-N-acetylglucosamine--LPS N-acetylglucosamine transferase] [protein_id=WP_042229788.1] [location=complement(1271730..1272866)] [gbkey=CDS]
MRKKRVLLLSEGFGAGHTQAAYALSENMRQLYPGIQTRVLELGAFLHPTLARWIFSAYRKTLHSQPKLYGKLYRFQYKKS
LNPITQMALHRIFYAQTAQIITQLRPDAIVCTHPFPSIVVSRLKRAGLSVPLCTVITDYDVHGTWISPEVNKYLVSSDKV
KTKLLHRGVAGDKIEITGIPVHPGFRQPHDKTEIRHRFELASIPTVLVMGGGWGLIGGDTVLDEMLRWRDRVQLIFCFGS
NQKALAKCQEDPRFLHPHVKLLGFTNEVDKLMEVSDLLVTKPGGMTCTEGLAKGIPMLFHKPIPGQEEENLLFFTENGFG
ESILSEETIRLWFKGLSERGNEMRRMREDIRESCRNMFQTDYSQVIMTLIHKEEQQSV
>lcl|NZ_CP026520.1_prot_WP_042229787.1_1095 [locus_tag=PC41400_RS05780] [protein=phosphatase PAP2 family protein] [protein_id=WP_042229787.1] [location=complement(1272922..1273449)] [gbkey=CDS]
MSKVVHWMHRHETRMFLYVNQRFRHTILDGILGTLTHLGGATATIAVTLLLSLFGKGVWQIAGIQALAALAISHIPVAIV
KKLYPRRRPYLVVPNTRTGRKPLQDHSFPSGHTTAIFSVIVPFVAHMPVLGFVLIPLALLVALSRMYLGLHYPSDCLAGA
VIGSGAAAASVALWP
>lcl|NZ_CP026520.1_prot_WP_042229786.1_1096 [locus_tag=PC41400_RS05785] [protein=glycosyltransferase family 1 protein] [protein_id=WP_042229786.1] [location=1273789..1274961] [gbkey=CDS]
MRVALFTDTFLPDVNGVAKTLGRWVRYLESRGAVCRVFAPAAGEAGTSGIRSVERFYSIPFSLYPDCRMAIPNPLLVGRM
LKEFRPTLIHLATPFNLGLTGLHYGKKHRIPLVASYHTHFDQYLAHYRLQWMEPMLWKYLLWFHQNCVKIYVPSPSTGEI
LHTKGLTGIELWGRGIDTAEFRPHTDRRKVLNKHNLLPGKFVLLYVGRLAPEKGLDALVRAFANLPEAFRSKCALILAGD
GPMRAELEKRLAPYPDVRFCGFVEGRELAELYAAADVFVFPSASETFGNVVLEAMASGTAVVGVNAGGVADNVRHGNTGL
LCPPDDASALAAAVQRLYEAPELRRKLALEGRSHALTRSWDSIFATLYESYREVAGDAGGGARRAAGIVK
>lcl|NZ_CP026520.1_prot_WP_042229785.1_1097 [locus_tag=PC41400_RS05790] [protein=glycosyltransferase family 2 protein] [protein_id=WP_042229785.1] [location=1275375..1276604] [gbkey=CDS]
MLDTILLVLQIVLVLFGAYQLFLTFFGWKRPKKKTQHPPQKSFAVLVAAHNEEEVVGALLENLKQLDYPKELYDVFVICD
NCTDSTADIVRSHGLYACERNNPNLRGKGYAIEWMLKELWKRPRQYDAVVMFDADNLVGHDYLRHMNDDLCSGSQVIQAY
LDTKNPNDSWVTASYAISYYFANRFWQVPRTNLGLANYLGGTGMCFESNLLKQIGWGATSLVEDLEFSMRCVKLGIKPTM
NYYAKVYDEKPLTFRASVKQRLRWMQGHFDVCRRYLFPLLWQGLKERSWTKIDAAFYSANVYNYFFGSILSVAIWIKAIV
PGGTDMNLYELAPIVNVLAISIYVMLPISLFIEKASGKTYKYLIAYPIFMLSWWPITVYAFFTQNNKQWSHTKHTRVIRL
EEVQSKQVS
>lcl|NZ_CP026520.1_prot_WP_042229784.1_1098 [gene=infC] [locus_tag=PC41400_RS05795] [protein=translation initiation factor IF-3] [protein_id=WP_042229784.1] [location=1276878..1277378] [gbkey=CDS]
MVNDEIRAKEVRLIGPDSEQLGIKPIREALQMAIDANLDLVNVAPTAKPPVCRIMDYGKYRYEMQKKEKEARKNQKIVDI
KEIRLSATIDEHDFQTKLRNAVKFLNDGDKVKASVRFRGREIAHAENGRKVLQRLAAETEEISSMERAPKLEGRSMIMIL
TPKVTT
>lcl|NZ_CP026520.1_prot_WP_009672104.1_1099 [gene=rpmI] [locus_tag=PC41400_RS05800] [protein=50S ribosomal protein L35] [protein_id=WP_009672104.1] [location=1277439..1277639] [gbkey=CDS]
MPKMKTHSSLKGRFKITGTGKVKRYKAYRNHLLSGKSGRQKRVLATNPVMAPGDVRRLKQGLAQIK
>lcl|NZ_CP026520.1_prot_WP_042229782.1_1100 [gene=rplT] [locus_tag=PC41400_RS05805] [protein=50S ribosomal protein L20] [protein_id=WP_042229782.1] [location=1277684..1278043] [gbkey=CDS]
MARVKGGFTSRRRHKKVLKLAKGYFGSKHRLFKTANEQVMKSLFYAYRDRRTKKRDFRRLWIVRINAAARVNGLSYSKLM
HGLKLAGIEVNRKILADLAVNDAKGFADLATAAKQKVNA
>lcl|NZ_CP026520.1_prot_WP_042229781.1_1101 [gene=ilvB] [locus_tag=PC41400_RS05810] [protein=biosynthetic-type acetolactate synthase large subunit] [protein_id=WP_042229781.1] [location=1278610..1280352] [gbkey=CDS]
MNVGTAKRSKEELMEAMRKPELINGSEILLRSLVLEGVECVFGYPGGSVLYIYDAMHGFSDFNHLLTRHEQGAIHAADGY
ARSTGNVGVCIATSGPGATNLVTGIATAFMDSVPLVVITGNVATTAIGTDAFQEADITGITMPITKHSYLVRDVNDLPRI
IHEAFFIANTGRKGPVLIDIPKDVSAQTAMFHPASEVNIRGYNPTVHPNKLQVEKLIKAIEEAEQPLIIAGGGVVYSNAS
KELLEFINKTQIPVTTTLLGLGGFPSAHELWVGMPGMHGTYTANQAIQNADLLISIGSRFDDRVTMKLDGFAPKAKKIAH
IDIDPAEIGKNVKTDIPVVGDIRSVLEYANTKAKPAKSEKWIERLQESKRNFPLKYKDSDTVLKPQYVIEMINETTKGEA
IVTTDVGQHQMWVAQYYKFQHPRSWISSGGLGTMGFGFPSAIGAQMGNPDKLVVSINGDGGMQMCSQELAICAINNIPVK
IVVINNQVLGMVRQWQEIIYDNRYSHIDLAGSPDFVKLAEAYGVKGLRATNKEEAQKAWQEALDTPGPVLIDFVVAKHEN
VFPMVTQGSTISEMLMGEDD
>lcl|NZ_CP026520.1_prot_WP_042229780.1_1102 [gene=ilvN] [locus_tag=PC41400_RS05815] [protein=acetolactate synthase small subunit] [protein_id=WP_042229780.1] [location=1280354..1280842] [gbkey=CDS]
MITKHTISILVNNQPGVLQRVSGLFGRRGYNIESITVGESEEVGLSRMVIVTEGDDNTLEQISKQLYKLIDVIKVIDLSK
NPMVARELALIKVNSEPAQRPEILGIVETFRAAVVDIGPSSLIVQTIGDSEKIDAMIELLRPYGIRELSRTGVTAMIRGA
IK
>lcl|NZ_CP026520.1_prot_WP_042229779.1_1103 [gene=ilvC] [locus_tag=PC41400_RS05820] [protein=ketol-acid reductoisomerase] [protein_id=WP_042229779.1] [location=1281064..1282056] [gbkey=CDS]
MAATMYYEKDADFASLQGKTIAIVGYGSQGHAQAQNLRDSGLNVIIGLRQGKSWNTAQNDGFEVFTVEEAAARADVIQIL
MPDETQARVYKEAIEPNMKKGVALMFSHGFNVHFGQIKAPADADVFLVAPKSPGHMVRRTYVEGFGVPGLIAIEQDATGN
AKQIGLAYAKGIGCTRAGVIETTFKEETETDLFGEQAVLCGGASELVKAGFETLVEAGYAPEMAYFECLHELKLIVDMMY
EGGLSSMRSSISNTAEYGDYVTGPRIITAETKKAMKDVLTDIQQGKFARDFILENQSNNAFMNATRRNEAEHPLEQVGSQ
LRELMHWIKK
>lcl|NZ_CP026520.1_prot_WP_042229777.1_1104 [locus_tag=PC41400_RS05825] [protein=2-isopropylmalate synthase] [protein_id=WP_042229777.1] [location=1282107..1283654] [gbkey=CDS]
MRKIYVFDTTLRDGEQSPGVNLNMQEKVEIALQLEKLGVDRIEAGFPAASPGDLEGVASVARAVRNATIIGLSRSREQDI
DAAREALRHAQDPCLHLFLATSPIHRKHKLRMEKQQVLETAEAAIRYAKKYFDKIEFSPEDAGRTELDFLCEVTEMAIRA
GATVVNIPDTVGFLNPSEFGNIFKTLKENVPGIEKIQLSAHCHDDLGMATANALAAVLNGADQIEGTINGIGERAGNTSI
EEVAMALETRQEFFQAKTSLVLSEIYRTSRLVSKLTGMDVPGNKAIVGANAFAHESGIHQDGMLKEKTTYEIMSPETIGL
KESKLVLGKHSGRHAFREKLVDMGYDLGDEQVNAAFAKFKDLADKKKNVLDDDIRALLEEKLVSTPEVYVLDSLQVSYSN
QAVPSASVRIHAEEGKVLQESADGNGSVDAIYNAIDKTTGEEVELEDYTIKSVSYGKDALGEVHVILGQKGVSVQGRGIS
TDILEASAKAYIDALNRLIEKRKTPTAKRSNVTLI
>lcl|NZ_CP026520.1_prot_WP_042229776.1_1105 [locus_tag=PC41400_RS05830] [protein=PLP-dependent aminotransferase family protein] [protein_id=WP_042229776.1] [location=1283722..1284927] [gbkey=CDS]
MNYRFSKTLGAFSSSAVREILKLTQGKSIISFAGGLPAEEFFPVEAVKEAIGRVMEGGKSALQYGLTEGYTPLRQSICNR
MATKGMKVTPDDMLLTTGSQQSIDLLTRVYIDEGDVILVERPTYLAAIQVFQARGAKIVSVRSDRHGMDLEDLAVQIKQH
RPKMVYVNPSFSNPAGRVWSLERRKGLLAACRQNDVLILEDDPYGELKFGTTEIMPSIFSLDQHPEGSCVVYTSTFSKIV
APALRTGWVMGDRAVIRNMAQAKQAADLHSSTLDQMTLHQLMEQFDMDAHIGLISVEYEKRMRTMVQVLQEKNWPGVHWN
DPQGGMFLWLELPAHIDTQVLLAAAVQQGVAFVPGVSFFADEPERNTMRLNYTHAGEAEIRLGLDRLDAVLQPLLAAAPR
V
>lcl|NZ_CP026520.1_prot_WP_042229774.1_1106 [gene=leuB] [locus_tag=PC41400_RS05835] [protein=3-isopropylmalate dehydrogenase] [protein_id=WP_042229774.1] [location=1285236..1286315] [gbkey=CDS]
MSDVKKIAVIAGDGIGPEVVAEAEKVLKKTEELFGHRFETEHGLFGGIAIDEKGTPLPQETLELCQRADAVLLGAVGGPK
WDNNPKELRPETGLLGIRKALGLFSNIRPAVVFDCLKDASTLKPEVLDGTDLIVVRELTGGIYFGEKFRRETENGQEAVD
TCAYNVSEVERIARQAFEIARTRSKKLASVDKANVLETSRLWRETVIRISADYPDVELEHVLVDNCAMQLLRRPSSFDVI
VTENMFGDILSDEAAMLTGSIGMLSSASLGEGSFGLYEPVHGSAPDIAGQGISNPIATILSVALMFRLTFGYHEAADAIE
RAVKEVLDAGHRTGDIAVDKLKAIGTKEMGDLIVAAIRK
>lcl|NZ_CP026520.1_prot_WP_042229773.1_1107 [locus_tag=PC41400_RS05840] [protein=peroxiredoxin] [protein_id=WP_042229773.1] [location=1286621..1287169] [gbkey=CDS]
MAERLVGKPAPAFTMETALGDGTDFGTASLSDYKGKWLVLFFYPLDFTFVCPTEITALSEAASTFKDLKTEVLGVSVDSK
HSHRAWINTAREDNGLGKLNFPLAADLTKSVARDYGVLIEEEGVALRGLFIIDPEGELKYQVVNHNDVGRSVEETLRVLQ
ALQSGGLCPMNWKPGDKNLSVK
>lcl|NZ_CP026520.1_prot_WP_042229772.1_1108 [gene=gcvH] [locus_tag=PC41400_RS05845] [protein=glycine cleavage system protein GcvH] [protein_id=WP_042229772.1] [location=1287263..1287649] [gbkey=CDS]
MDIPQELLYTPNHEWVRVDGSRATIGLTDYAQMEIGMIVFAELPEEGEELQAGDFLGIMESVTAETDVFMPLSGTIVSTN
RELVKNSHYINRFPYGSGWLIQIEMNKPEELEALWSAEKYAEVFEERL
>lcl|NZ_CP026520.1_prot_WP_042229771.1_1109 [locus_tag=PC41400_RS05850] [protein=glycoside hydrolase family 15 protein] [protein_id=WP_042229771.1] [location=complement(1287953..1289917)] [gbkey=CDS]
MPRHFVVGNGKFLINLDEHSYMRDLYYPYVGQLNHIGGYHCRVGIWVDGAFSWLDAPEWQFQRAYVEDSLVTDVKATHPG
LGVTLRIQDGVHQREDIFLKRIQITNHHSAVKEVRVFFNQDLVINETEVGDTAAYYPANHTVFHYKKDKYFMFNGSTGTD
GIFQYTTGVKRFYNAEGTWRDAEDGHLMGNPISQGSVDSTISFRMYVPPSGDHTLYYWLSAGRTLEDVKKLDKYVKESHP
GKLLDRVQIYWQRWVNKTERSFGDLSGEIIRLFKQSLLLVRTQTDVNGAIIAANDSDIMQSNRDHYSYMWPRDGALVAYS
MSMAGHQAMMEPFYRFCANVISPEGYLHHKYNPDGTVGSSWHPYVHNGGEQLPIQEDETALVLFALWQDYKKHGNLELVQ
SLYRDLIRKAARFMSTYIDQELNLPKPSYDLWEERYGIYTFTASAVYGGLIAAANFCNLFGDEERSSRYRHTAEKIKNGI
LTHLWDEKEGRFARGLHQKDGTWVRDMTLESSVYGIFEFGVLPADDDRVVRTMQANKNGLSIKTPVGGVARYHHDYYFQR
SQDVNQVPGNPWIICTLWIAEWEIECAKKLEDLEAPRQTLEWVVKHQMESGILPEQLDPYNGDPVSVAPLTWSHATFVLA
VVKYLSKYEQLAGK
>lcl|NZ_CP026520.1_prot_WP_042229769.1_1110 [locus_tag=PC41400_RS05855] [protein=DUF1957 domain-containing protein] [protein_id=WP_042229769.1] [location=complement(1289991..1293545)] [gbkey=CDS]
MSVSQPVSALLVFHAHLPYVYRPGSGLTLEEYWFHEAMCDVYLPFADCMDRLLNGKVCSAVTLSVSPVLLSMWENPDLQQ
RTREILKRRVELGHKEKERLADKPGPASLASRYAGRYTDMLALFDKIDGKIIPRLRLYADHGLIQLIPAAATHPFLPLFK
TEEAAFAQLYTAVTEFRRYFGFAPSGIWLPECGYTPMLEPILLRLGLRFTILEAPEDKAGDSGQGPAAPYVTPDGLRVFT
RDAAASALVWSRDSGYPGHPDYREYYRDIGYELGRGTPAEQAYIRPYVLPDGTSVATGYKYCRITGAAGGADEAGGKAPY
DAGRAAELAAVHARHFLASLGAAEPATTGGAAVDAAGSAADWAAAVPTVQAAAGLASPASAAAKRQPAEAPPASAPAGDA
ARQTRAELSDAELRYDRQASCAAPEAEPVRLSASYAGESGARPQAASVSEARLLPPRPVAVCAFDLELFGHWWHEGPLWL
ETLLREGDREQRRRYTASGSHSGPSLPDSEKQPVFLTPSLYEKRYAFGKGTAPVLSSWGRGGDARVWLNPATDWMYPLLH
AAEEAMALAAARAARTDTAARKAHQALNVELDAISSSDPFSFYSSDPGPNRRDDENLSHTAITDRILRQAARELMLAQSS
DWAFMIEAGEMAGYAAGRFHEHRNNFYQLLPILDDIDERQPAVLSPEKLELLHRMERSASLFPDVNWRAYLTGYLSETLA
GKKRSQAPASALAAPLLEEQTAAQESETGKPSAGSSQAKPPAANSFTTGPNTGKKTSLRVLVLAWEFPPHLIGGLGRAVG
DLTAQLARDGHTVHVITCRRDDEASDSLWNGVHVHRVEVLQSPESPAFLDWVFRLNIAMILRMERLLEENQELRFDILHA
HDWLVYYAARDCRQALRLPLVATLHATEHGRKLGRLDDPVSRKIHAAESSLAELADKLIVCSDSMAEEVTSLFNLKKSGV
TVIPNGLPAASDAESSRKPHPAPACAGDSGRTLRSAFAKRRILYLGRLVPEKNVPLLLAAFPYVLEQFPETELVIAGTGP
EAASLSGLAAPFGDRVRFTGFADEAAKAELLAEADVCAIPSLYEPFGIVALEAMRAGVPLVAADTGGLAGLVEDGVDGFR
LPPADPRAWAGKICLLLGDPELAALFTKRASAKLHTCFNMKEIGAATANVYQSAINCQRSTKTC
>lcl|NZ_CP026520.1_prot_WP_042229961.1_1111 [locus_tag=PC41400_RS05860] [protein=DUF4912 domain-containing protein] [protein_id=WP_042229961.1] [location=complement(1293542..1293991)] [gbkey=CDS]
MIQRQTVLFVYWSISSRLAPMIAEYGTEAEQGLVSVRINGYGESSDESVVYEVRTMPPAENAGSAYIRTQGRKGLWQAEL
GILNANCGFLPLLRSAPVRTGDGSGESPFPEEGRHSGNNHPPFPLLPVDFGFYSTYTIYPPPKQEGEKP
>lcl|NZ_CP026520.1_prot_WP_042229768.1_1112 [locus_tag=PC41400_RS05865] [protein=hypothetical protein] [protein_id=WP_042229768.1] [location=complement(1294237..1294452)] [gbkey=CDS]
MTIIPIISRQIAFAKFESQTGEFSVYYHTGKVENYTGFNLEDIQALWDSANRYDTLMRLTESRRLPQTTPT
>lcl|NZ_CP026520.1_prot_WP_042229766.1_1113 [locus_tag=PC41400_RS05870] [protein=hypothetical protein] [protein_id=WP_042229766.1] [location=1294771..1295196] [gbkey=CDS]
MSFCCGASMIGSKGTLKHIRTSIHNVPILFCPVCHRIEVHHLIENEYELLAEYAHGDGAAEVDFIDYVEGKQGPAIYENC
VNHEDEEPMDVVQNQIDMGLDLLSVAKSLGDSAWEDQLKKRLQVLSRRRSKLVKGRTPQSP
>lcl|NZ_CP026520.1_prot_WP_042229765.1_1114 [gene=sigI] [locus_tag=PC41400_RS05875] [protein=RNA polymerase sigma factor SigI] [protein_id=WP_042229765.1] [location=1295568..1296389] [gbkey=CDS]
MILVLFKKLFGSKRQSSHHQKNAPQPEEVVKKIQQGDLRLRNQFISDYQPFVIKVASRFCKRYIDPARDDEFSIALAAFN
EAINQFSHDMGRSFLGFAETVMRRRLIDYLRKEQRFKGQIPYSAFDQEDEEDHYVNPIEIHQALEQYEKQKGIEERRGEI
VEFNRNLAVFSIQFTDLVDASPKHEDSRIMLFGIGKLLSEDPELMRLLLSKKQLPIKELLGKVSVSRKTLERNRKYLIAI
ALINTGSYPYLKEYLHIKEQSKQTFQERREAYE
>lcl|NZ_CP026520.1_prot_WP_042229763.1_1115 [locus_tag=PC41400_RS05880] [protein=anti-sigma factor domain-containing protein] [protein_id=WP_042229763.1] [location=1296382..1297695] [gbkey=CDS]
MSKGIVMELKGNSIIVMTEEGRFERISKKGRNCQVGQEIHYKSTRGFGRSQILVLTSGLTAAAIICFVFIGGLLNMGGQV
KVAAYVTMDVNPSVELALDDEEIVREVRGLNQDGARMAKALHVEGKPLTQAASVILDWLDQGPLAKGEGDIIISSTVIGN
ERKVSDVLLADKMEKQVTDHLLANHADLSKYAVAAFAAPPELRQEAGTHQLSTGKYAIYLNAKSMGTEISLDQLREMSIY
TLAKQMGGMGKLYNRTQPINKDNLTNWLAMEKEGSWNKDNMALPNEEEKNQPEATKTPESTPTPTPPNNAAGNTANTSGK
TSGTQNTPVPTPTPKKENTYTGSGNADNAREEWLRQKVEREYKRAQEELKRKQEAQKEKEERERKAREKEREKQQKEKEK
KEKNRENDDDRDDQGSWKGGHSFGVPFKEGPVYYNIS
>lcl|NZ_CP026520.1_prot_WP_042229762.1_1116 [locus_tag=PC41400_RS05885] [protein=ATP-binding protein] [protein_id=WP_042229762.1] [location=1298047..1299405] [gbkey=CDS]
MPDIRTMYLQVLQVLMQSGPEEARSYSSTLKPLISSMPPQEVTLLHDRCMRELTGPARPEDVVLYYQQSFQILSEAISFC
TSQSEQVGREPSRRFTDRLMQDSSYPPSNSKYENVLQHMDSGIALFDQEGFLSFANVKLGQFLGLPRKSMIGRTFYELVR
LRKLSRGMRKLLIKLHREMFVYRLPFQEVLDEKGQHFLVTAKYDEELDGDILISVKDVTDFKRIEQSVYYNDKLAMLGKI
AASIAHEIRNPLTSIRGFIQLLQPELLRLGKQDYADIILSEIDRANDIIFEFLNSSKPTAPVKQKIQIASLIKEVVLLFE
SEALLKGCDIQLSESDPNLYVWVDVKQIKQVLLNIVKNAIEVMSDGQKEHGLIRIRTIKVDHQVWIYIEDNGRGMDQKTL
SKLFDPFFTTKEDGTGLGLSVCYRIIKNHGGVIHVESEVGEGTSFRIQLPLN
>lcl|NZ_CP026520.1_prot_WP_042229761.1_1117 [locus_tag=PC41400_RS05890] [protein=leucyl aminopeptidase] [protein_id=WP_042229761.1] [location=1299576..1301306] [gbkey=CDS]
MRADYTTNTKISIAESGGWASADADALLFFTTEGEDPGSLPDLPEELRAELDRLKRAGRYKGSAGEVTVLPAYGSLAAPY
VIAAGLGAAPAGRDTLRLAAVQAAREAQRLGLGRLAVLLPEGAFPAEPPRGGTYAAPLAAQGTGGADAARGASVKAASVD
GVQEEPQSVSGADARTAAGAPSTGSADERPAGAASAAKTVSTVAEGLLLGAYRIATYAREAKPRQELAAAQLFVQRDALN
GEALQAAVAAAEAYAVATNYARDLTNLPGNLLLPSDLAKEAEKLAGQFGIACEVLDEQAIVERGMGALAAVGLGSANPPR
MITLSYDGDPSSGEVLGLVGKGVTFDTGGISIKPAGGMEEMISDMGGAAVLLGLLIVVGRLKPRINLRIVIPAAENMPSG
TAMRPGDVITTLSGRTVEVLNTDAEGRLILADGVTYAIEKGATRLIDIATLTGAVLVSFADVATSAQTNDDAFLASVLQA
SIQAGEKVWQLPNFPEYREMLRSTVADIKNAAAHKWAGAIMGGSFIEAFIEDKPWIHLDTGGTAWLWGDRGIDPKGGTGA
MVRTLAEYICGAGHGA
>lcl|NZ_CP026520.1_prot_WP_042229759.1_1118 [locus_tag=PC41400_RS05895] [protein=general stress protein] [protein_id=WP_042229759.1] [location=complement(1301543..1302142)] [gbkey=CDS]
MDKKIVGVFDTEREAIQAIDGLKQQGFHADDISVIAKDKQDSADILDETGTKAPEGVATGAATGGILGGVTGLLAGIGTL
AIPGVGPFIAAGPIAATLTGAAVGAGAGGLVGGLVGMGIPEDEAQQYDNYVNDGKILVLVDSDAERDAHVYDTFRTNRSL
NASTYDDQVFRDTDTPAYDVNSGLPYNAPNDDIDRSRRL
>lcl|NZ_CP026520.1_prot_WP_084706588.1_1119 [locus_tag=PC41400_RS05900] [protein=hypothetical protein] [protein_id=WP_084706588.1] [location=1302465..1302680] [gbkey=CDS]
MKLKLRKADEMERYQTDKSAKIAFIFYTGALLAWGLYNFFSTGKTGGEFTILLVGNAIFLWSRVVYRRKMQ
>lcl|NZ_CP026520.1_prot_WP_042229757.1_1120 [locus_tag=PC41400_RS05905] [protein=S-layer homology domain-containing protein] [protein_id=WP_042229757.1] [location=1302843..1303877] [gbkey=CDS]
MKRSFWKKSALAAGTALVLVTAASPASAHWLYEDIDGVPWAQDSLISTSQMRVIDGVSEKTFDPGAKVTGIQFVKMAVMA
NDPSYEPDLGSYAAWYGAYLNKAQAEIGLVDEAFQKTMLDPASRLEIVRVVSKLIQPEWRFKSVPDEDAVALCVKEGILQ
GRAEGDLALEAPVTRAEAAVLIDRVMEKTGKYGKRFDKPVVGVEGGAFVLNGLYLGQKAEEVTAILGTPFLKGQDELDGD
AAEEYKNFYIAYYEGNASRIMYKAVDEKLQSGDVELPGAAVFKGFDSTYYYLEQSEEILMVKDQSTYLGFADGNFYFHVD
NGGIQPYNEAARKLNDLFEKHKNK
>lcl|NZ_CP026520.1_prot_WP_042229755.1_1121 [locus_tag=PC41400_RS05910] [protein=Ig-like domain-containing protein] [protein_id=WP_042229755.1] [location=1304277..1306862] [gbkey=CDS]
MMRAMRILALFLTLTLVSSSFGAMSAMALEVTSLVLNKNELSLEVGGTASLTATAVYDSGATENVTVKTDWTSGSPEIAS
VYAGSVTAKKEGKAVITATHKGKTVIVNVTVSKHVRSLIKDVQTIDLRKGKSEQIKLTAYYDDGTNEDVTKKAEWNIDNG
SVATVVNGLVTGQSSGTAIVSAKYNNQTVSVPVSIEIAKRVDPDKSQISMLLKDSEAITLIATYPDGTSEDVSDIAEWES
DNPNVADVLKGKVTGYGPGKATIKASYGTKSVTIKVDVDNAIKLDLGKETVLLKKNGTEQLNLKATYADGSSDDITARAD
WSSSDENIVSVVKGKLFANATGEVTVTAKYGTKTVSVVVDVDVPKRLEAEKDPLFLQTGKEEQIVLWATYADDTKEDVAD
EAKWTVDNEAIVSVTDGKVTARKAGEANVTATYGGKTVTVKVSVDIPNTIKASVKTVNFQVGSYEKVTLNAIYSDGKVED
VTEKADWTSSAADIAEVRKGVITGVGTGAATVTAKYGTRTAVVQVSVGVLKTLTTGSKTTISLKKTGTQVLDLTATYSDG
TKEDVTKEAAWTSSNEKAATVDAGLVTGVASGESTITATFNNKTVTFKVQVDMADKLTANPTFLNFDLGENKLITLTSTD
GSGTPKDVTKEAEWTSSSAAVATVDNGLVTPVSRGKTTITAKYGGKTVTIPVEIGVVSTLEVDKRFVTTKSGSSVQIKVT
ATLSDGSKKDVTSSVLWKSSNYKIADVSAGKVTGVSAGSTTVSATFGGKSISIPVDVDKLKYLQTDKVTVELKAGATANV
KATATFEDLSEQDATVQALWKSSNIRVADVKDGIIRATGKGKATITITYAKKSTKVYVTVN
>lcl|NZ_CP026520.1_prot_WP_156972819.1_1122 [locus_tag=PC41400_RS29655] [protein=hypothetical protein] [protein_id=WP_156972819.1] [location=1307002..1307178] [gbkey=CDS]
MFVTTPFNVACGSCRYSNHELYTASIGNPDPYGDISGFFGQVKEKDQVPPAPENLVFF
>lcl|NZ_CP026520.1_prot_WP_042229754.1_1123 [locus_tag=PC41400_RS05915] [protein=SDR family oxidoreductase] [protein_id=WP_042229754.1] [location=complement(1307183..1307902)] [gbkey=CDS]
MTLAGKKIVIIGGSSGIGLETAALALEQGAEVVIASRSADKLRQAHNTLGSSVSAYELDMADEQAVKEFMTRDVKRLDHL
VVTAAQTWSGPFLATETSKARELFDSKFWGQYYAAKHAAPLLAEGGSITLFSGLVAFKPMAGAASLGAVNAAVASLGQTL
ALELAPLRVNVVSPGIIDTPSRAGMDDAARSAFYESVAARLPVKRVGQPGDVAQSVLYLLHNSFVTGTVLHVDGGHLLT
>lcl|NZ_CP026520.1_prot_WP_042229753.1_1124 [locus_tag=PC41400_RS05920] [protein=helix-turn-helix domain-containing protein] [protein_id=WP_042229753.1] [location=1308114..1308443] [gbkey=CDS]
MEEVKLSKGTPGTKCPIEAALGIIGSKWTFLIIRDLLIDGTVRFGELLKSLEGISPKTLSVRLRELEESGIVVRTVYPEV
PPRVEYTLTDKGKQLESIFIELKVWGLKL
>lcl|NZ_CP026520.1_prot_WP_241688796.1_1125 [locus_tag=PC41400_RS05925] [protein=GGDEF domain-containing protein] [protein_id=WP_241688796.1] [location=complement(1308629..1310131)] [gbkey=CDS]
MSTLRELIFPSKQTGDSLLFAQSHHGHLFHRIKLVSILLIAVYPYYFYADFFLYLDVPVEAYRYGSIGIHLLSTLLSVLF
LILYRLLRNQTAASRTVWPRLLVTIYTALYLFIGMLSSINSQSLTGNVDAYIIILLSAAIILPIHPKNFLVILLINHPVF
LVLLSMANKDKFDLLTKQINTTGTAIIACFIVLTLYNLRKKDFINTRKLQENEIHFRKLFSVNPYPLVLSRIRDNTVVQL
NAEAAHFFQIPLDSGTPADASFTYPSSAEKEEIIHELLQKDSIKNRICQLSVSPHSQKWVMINYELIIYDNETCVLAGIT
DITELKKAEAELARYASLDALTGALNRGHGMALLQQELDQCREQSGELVVCFVDINNLKEVNDRYGHAEGDSLINKACEV
LQRNLEPEDYLFRYGGDEFVLVLHTPPDAAELIWTRIREDLLALSREPDIPFPVSVSHGFFAVGPGTDMSAAEIIEAADK
EMYIEKHSYKNNTHSPSNTM
>lcl|NZ_CP026520.1_prot_WP_196809670.1_1126 [gene=rpoD] [locus_tag=PC41400_RS05930] [protein=RNA polymerase sigma factor RpoD] [protein_id=WP_196809670.1] [location=1310530..1311441] [gbkey=CDS]
MKPGKDDSREAEQGGQQDDLSLPHGIRMDDPVRMYLKEIGRVGLLSAEEEVNIAKRIEQGDEEAKRILAEANLRLVVSIA
RRYAGRGMLFLDLIQEGNMGLMKAVEKFDYTKGYKFSTYATWWIRQAITRSIADQARTIRIPVHMVETINKLSRVSRELL
QDLGREPAPEEIAKEMELSVEKVREIMKIAQEPVSLETPVGEESDTNLGDFIEDQDAMAPADAAAFELLKVQLEDVLDTL
SEREENVLRLRFGLDDGRTRTLEEVGQVFGVTRERIRQIEAKALRKLRHPSRSKRLKDFLDES
>lcl|NZ_CP026520.1_prot_WP_241688797.1_1127 [locus_tag=PC41400_RS05935] [protein=hypothetical protein] [protein_id=WP_241688797.1] [location=1311461..1311715] [gbkey=CDS]
MTAGFVIRCLNCGEEHKVVPGQRAYRDERIVFGTDNYGADQISCECGNALEEHSEYEGEAFHSEFVVKWRDLSGKEPVNR
TNNI
>lcl|NZ_CP026520.1_prot_WP_042229749.1_1128 [locus_tag=PC41400_RS05940] [protein=oligoribonuclease] [protein_id=WP_042229749.1] [location=1311915..1313105] [gbkey=CDS]
MIHLYTHNDLDGVGCGILAKCAFGEEAEVRYNSVEGLNVQVQKFLELVRQGKAKHEYLYITDLSVNEQNEQELNEYAQSG
GKISMIDHHKTALHFNAYPWAAVTVAYPDGRLASATSLFYEHLLRNGFLKATPALDEFVELVRQYDTWEWEANGNAQAKR
LNDLFYLVSIDEFEEKMVPRLLGSEPFAFDDFEEKILDMEESKIERYIRRKRREIVQTFIHDYCVGIVHAESYHSELGNE
LGKEYPHLDYIAILNMGGKRVSFRTIHDHVDVSQVAGRYGGGGHAKASGCSLTDEVYPAYVAEPFSVEPIRTDAFRNYYN
VKNSSFGTLYEGRSDTRFFLYPFDGKWKMEINGEEVDASYETYEEAERFIKRHYAAWLVRDDAFVKFLMKHFRKET
>lcl|NZ_CP026520.1_prot_WP_042229747.1_1129 [locus_tag=PC41400_RS05945] [protein=NtaA/DmoA family FMN-dependent monooxygenase] [protein_id=WP_042229747.1] [location=complement(1313305..1314609)] [gbkey=CDS]
MITHRPLCIGLSLSATWMKGDGWRRPGSGVEKMDSIDYYIELAKTAEKAKLDFLFKADYLYVSPQMLGDSANFGSPDPTM
IFAAIARETERIGLVTTVSTTFNPPYVVARQLQSLHWLSNGRAGWNIVTSIEGAENFSDSPMPSPQERYAKAMEFTDVVR
KLWESFPQEAIVMDRESGIFSDKDKISPIHHSGEFFSVQGPLTLPAHKSGSIPLFQAGASDTGRNFASSVADAIFAAMPD
LESGVELRNDLRRRAREHGRNPNAIRVLPGLYFFLGETREEAGELHKAAHANLSMERRLASLKSVLGLDLSGFTSEHRVT
AEMLPDPDQPVRSRTHAELLRRYITNRQPTVEEVLARPEVVGSAHWVSVGTVEDVLNDIIERFEAGAIDGFIALPGGSEK
SMELFFEKLMPQLVEKGLFRSEYAGSTLRAHLGI
>lcl|NZ_CP026520.1_prot_WP_042229746.1_1130 [locus_tag=PC41400_RS05950] [protein=AraC family transcriptional regulator] [protein_id=WP_042229746.1] [location=complement(1314763..1315590)] [gbkey=CDS]
MIDLDKLAGIFAGCSYDIEGTYRLVIQPQSVLSDFRTVRHGFLFIIRGEAKMCVNGTAYRLLPGSVFHAAPGMRLDSRVI
GQSELEYYLLFYRLDKLGNENRDHECDSHFKLEPGANPRVTELLILLQQNVHTPGGIGKLRVKELFLSIVHQVLAGCRYR
ENSHFPSKRVIEEAVAYINGHYMNPLTLDGLAELHAMSSKRFSYFFHKYTGFRPIDYVIRYRMERASELLKTGHFPIRDI
AVSVGYANQLYFSRIFKKKFGVSPSAYIHKMENYK
>lcl|NZ_CP026520.1_prot_WP_042229745.1_1131 [locus_tag=PC41400_RS05955] [protein=NAD(P)-dependent oxidoreductase] [protein_id=WP_042229745.1] [location=1315876..1316667] [gbkey=CDS]
MKRKILITGATGRIGSTLLAGLGNLDRYDITGADLEADEEAGIVRLDVTDADRLEELTKGADTILHIAWAEDDEDFLGKV
LPVNVTGAYRLYEAARKNGVKRVIFASSNHATGFYKVGEDVEVTDPYRPDSFYGLSKCYIELLGRLYADKYGISSFNMRI
GNFPGDDRPHSERSAHIWISSRDMVQLAERCIEADASMKYLVLYGTSANTGNYYDIGYLKELIGYAPQDDSAVLMEEARQ
RRETIKEDETPFQGGSFVDKKPS
>lcl|NZ_CP026520.1_prot_WP_042229743.1_1132 [locus_tag=PC41400_RS05960] [protein=chromate transporter] [protein_id=WP_042229743.1] [location=complement(1316858..1317406)] [gbkey=CDS]
MIEDWLNLLIGFFVSNVLGYGGGPSSIPLVYQEVVTHYHWADDRQFSNILALGNTLPGPIATKIAAYVGYDVAGWFGLTV
AIAATVVPSAIALIVLLKLLQKHRQSSVVKGMTLLVQPVIAVMMVVLTWQMAEGSVQSIGIWQSLGIAAVAYWAMQRRKI
HPAFVIIAAFIYGGVAIGYFGI
>lcl|NZ_CP026520.1_prot_WP_042229742.1_1133 [locus_tag=PC41400_RS05965] [protein=chromate transporter] [protein_id=WP_042229742.1] [location=complement(1317403..1318026)] [gbkey=CDS]
MEQASRGRKPYYELTSAMVKTGILGYGGGPSVIPLIRYEAVNHYGWMEDEEFGEILALANALPGPIATKIAAYLGYRQKG
TSGAIVAVIAHILPSSLAMIALLSVVGYLSSSAVVQGMIAAVIPVVAVMIGLMAYEFAERAVKGLGKPAGFGFFALSLLM
LEIIQIHAAIVIIVFLAYGAFHFKAVRWLRGRLGKQGAPDDGKEDGS
>lcl|NZ_CP026520.1_prot_WP_042229741.1_1134 [locus_tag=PC41400_RS05970] [protein=Lrp/AsnC family transcriptional regulator] [protein_id=WP_042229741.1] [location=complement(1318028..1318510)] [gbkey=CDS]
MNTQDPPETKSAGIDETDRKIIRALNANGRISYTDLAKDIGLSRVAVQGRIQALMETGVIERFTAVINPEKIGIGVSAFF
NVEVEPKFLHQVADELAEEPSVTSLYHMTGPSKLHMHGLFKNHHEMELFLKEKLYPLPGITSVDCQVLITRFKSRMGMRL
>lcl|NZ_CP026520.1_prot_WP_042229740.1_1135 [locus_tag=PC41400_RS05975] [protein=peptidase domain-containing ABC transporter] [protein_id=WP_042229740.1] [location=complement(1318951..1321137)] [gbkey=CDS]
MLRLSRKPVCIRQHDRKDCAPACLAAVARYYGLKMPLSLIREACGTDHTGTNAYGIVQGAGKLGFTAEAVKGDAPALFAG
FELPAIAHVIDREVQHYIVIYKITPDKVLAADPAEGMRTYKTEEFLNLWTGVLILLTPGESFKPGSRVTGAAKRFFPLLR
RNRKLLLPILAASLLVTAIGMAGAFYFQILLDTIMAEKLQGTLTLVSVGIIVLYAVQTLLDFARNHLFLYLAQRIDRSLV
LDYYNHVISLPMSFFSSRKTGEITSRLTDASNVRDALSEASLHIAVDSLTLLIAAAVLYNQNPLLFGLTLLLVPVYVLIM
AGFRGFFDRYNRKMMNENAELTSYVIESIKGIETVKASTAQREVKETARAKYEAFLATAFKYGLMNNVQSALKTALQLIS
GVVLLWIGARQVLAGEITAGQLITFNVLLAYFLSPLQRLIDLQPVLQSAYVAASRLGEYMDLAPEKELSPDESPVSGNAL
DLRGDVLIRNLSYRYGSRPLTLDDVNLTIREGSSTALVGPSGSGKTSLLKLLMKFFEAEQGGIEINGHDLRDIPAGILRS
RIAYVPQETFFYHGTVYDNLCLGLDPKPSLEEVAEVCKQAQIHDFIASLPQHYEAVLEENGSNLSGGQRQRLSIARALMK
KPHLLLMDEATSSLDAVTEKAVSSLTAGLGITTVMISHRLSGVRHCDRICVMDRGRIVETGSHDELMRRGGAYSALWNSQ
FAEAQAAL
>lcl|NZ_CP026520.1_prot_WP_009672156.1_1136 [locus_tag=PC41400_RS05980] [protein=bacteriocin-type signal sequence] [protein_id=WP_009672156.1] [location=1321324..1321545] [gbkey=CDS]
MKTLKSNHSFTKLDETEMMNVNGGDWRDHARWTADFLRGLFSNPLGCGLGDVTGGNSKPAKGCSPYPAKNSNC
>lcl|NZ_CP026520.1_prot_WP_042229736.1_1137 [locus_tag=PC41400_RS05985] [protein=glycosyltransferase family 2 protein] [protein_id=WP_042229736.1] [location=1321649..1323169] [gbkey=CDS]
MTTMDRDWLNAYRYNLPFFETIHKELLRSAEESRNPGPEHAFGEVRAQAEPGMPLPLSGSVHSTQPSSADGLAENLLRRV
EKRITLLEKVRECSDELASLIGEEPPHMREELAALNGSEYGELLEDLLRPGQASISVVIPVFNEERCIARCLDSVQGLAD
EIIVLDTGSTDGTVDIIRGGHPHVRIEFAEWQDDFAEIRNRLNDLAAGDWIFQLDADEALIGDGAELRAFLGLFYDCPAE
ALVVSPKIVNHDGGELTLTRRMYRREDGLRYYGVIHEDLRREMDERGADLRHLMTNFTLSHDGYEPKVIKAKNKHERNIR
LEEQMVSREPDNIRWHYFLAREKQEAGYPDHEVVEGIRKGLRLRGTRTEPDHFHLRSLLLLAGILDRQPQYNEPLNALAE
EINAIYPDCTDGLYYLLLHEYTDGTGNLVTRAESGLNRLKQLEHPYSLLHPRSDHVTRLIGEIFMAAGQYKRAFHLYSLI
SDERILQDLRERLGGLQEEIAAFLEG
>lcl|NZ_CP026520.1_prot_WP_174446801.1_1138 [locus_tag=PC41400_RS05990] [protein=amidase] [protein_id=WP_174446801.1] [location=1323359..1324807] [gbkey=CDS]
MVLFAEATIAGLQHAMEQGECTSFQLVRYYIERIAAFDQQGPGINSVLELNPDALHLAEAADAQRRLGQVRGPLHGIPVL
LKDNIQTADNMHTSAGSLALKDHYAGEDAFLVKKLREAGAIILGKANMTEWANFMADDMISGYSSRGGQVLNPYGADLFA
GGSSTGSGASVAIDVTTVAIGTETSGSILSPAIHNSVVGIKPTLGRISRSGIIPLAHSQDTAGPMARTVADAAALLSVLC
GYDPQDAATGSAAGLEEPDFTVHLDKDGLKGARIGVPRSEHLDPLTDEERELFEGLLEEIREAGAVIVDPVEIPSLEELS
GHHSKVFYYEFKADINAYLSRLPDGLPVYTLAELIAFNEAHADEALKYGQQVLIRSEATSGKLTEPEYLLHRLADLRLSR
TEGIDFAMRKHQLDALLFPNTAGDEIAAKAGYPSIAVPSGYLRSGMPFGVMFTGGAFAEPMLIKLAYAFEQLRPRRKPPV
LG
>lcl|NZ_CP026520.1_prot_WP_042229734.1_1139 [locus_tag=PC41400_RS05995] [protein=glycosyltransferase family 2 protein] [protein_id=WP_042229734.1] [location=complement(1325102..1326196)] [gbkey=CDS]
MPTISLCMIVKNEEDHIGRCLASVADLVDEIIIVDTGSTDRTVPIVREYTDRVHHFRWIDDFAAARNYSFSLAEKEYILW
LDADDVFEEPDRIKLASLKKTLEPSVDAVNMVYNLAFDEFGQVSASIRRNRLVKRSCGFLWHGPVHEYLEVGGSVRTTDI
AVTHRALYHDSGRNLRIYENRLARRETFSPRDLFYFANELKDHRMYNRAIHYYEEFLGTGNGWIEDNISACGRLADCFHA
LGDEKRELDSVFRSFLYASPRAEFCCRLGYVFLRKKEYRTAAYWYEAALQAGKHTDSWALTHVACSTWLPHLQLCVCYDR
LGEHELAHRHNETVLEYRPGDRQALANKTYLDGVLARKEEAPAD
>lcl|NZ_CP026520.1_prot_WP_042229732.1_1140 [locus_tag=PC41400_RS06000] [protein=FadR/GntR family transcriptional regulator] [protein_id=WP_042229732.1] [location=complement(1326604..1327335)] [gbkey=CDS]
MDVNTNGRPLKSHEWVLNDLRKQLEEGRLAVGDRLSSVVDLAAQYGVGRSTIREALSVLKAMGLLDIRQGGGTFVKALPT
EPPHPGLLRPEWVGRAESLRHILEVRRVLETGCASLAAVSRTDDDLAALAETLRSMEEHLGDEALSEQADVRFHEQIAAA
THNPVLADLMASLSQKLHDSMKDTRALWFYAERSSAERLLREHQAIFDAIRSRQPQEASTRMEQHITKVEQVLGEKGADV
PSR
>lcl|NZ_CP026520.1_prot_WP_042229730.1_1141 [locus_tag=PC41400_RS06005] [protein=FAD-linked oxidase C-terminal domain-containing protein] [protein_id=WP_042229730.1] [location=1327541..1328956] [gbkey=CDS]
MLNEQVRQKLIAIAGAEGFKDDPQSLVTHSYDGTPMLQSLPDGVIYPADTAQVSAIMKVLSEHRVPLISRGSGTNLCGGT
VPVQGGVVMVMHRMNRILEVDLENLTAVVQPGLNTKQFITHVESLGLFYPPDPSSMSISTLGGNIAECSGGLRGLKYGTT
KDYVIGLEAVLPSGEIIRTGGKLMKDVAGYDLTKLLVGSEGTLAVITEATLKLIPPPKAKKTMLAMYKDLYGAARTVSKI
IENRIIPATLEFLDNPTIRVVDDFAKLGLPLDMEAILLIEQDGDPATVERDIAMIENICRSENADRIAVAADEEEALKLL
TARRSAFTALARLRPTTILEDATVPRSKIADMVMEINRIAKKHDVTICTFGHAGDGNLHPTATTDARDPEEVHRVEAAFE
EIFEAAIRLGGTITGEHGVGLVKAPFLEWKVGAAGIEVMKGIKQAFDPFNLLNPGKVFAKESRKRVVIDRG
>lcl|NZ_CP026520.1_prot_WP_042229729.1_1142 [locus_tag=PC41400_RS06010] [protein=(Fe-S)-binding protein] [protein_id=WP_042229729.1] [location=1328949..1330517] [gbkey=CDS]
MAEKAACTTSNPLAQKLKLRLDEDQLTNCMRCGFCLPACPTFRETGIEAESPRGRIALMKAVADGIMAPDAAFEEQMNHC
LGCRACEPACPADVKYGQLIEQARDAIEDHTDSHRGWVKLARRTAFKAVFPKQNRMKLLGGALGFYQRSGLQKITRGSGI
MKLFPDHMREMEQILPESSSRGVIERIGSRHPAVGAKVATVGLFRGCLMDVMFTETNVHTVELLTQAGFEVVIPDTQNCC
GALHAHSGEADGARSLARGNIRAFREAGVDYIVSNAGGCGAILTEYDHLLHEDAQWKDDAAWFAARVKDVSDLLVNFGRV
PKFAERGRALTAGGPVAAGASAPQAADPAAQAEASAAVAGTPATAAAAGSTAAAPADIGAELAAIEDASVQPSAASAAAA
SAPASSVRITYQDSCHLRNVMRSSDAPRKLMRQVAGVEYVEMFEADRCCGSAGIYNLTQPEMANQILDHKMVHANATEAR
YLLTSNPGCLLQMKLGNERHGDGSMKVMHVVDFLHERLLGDR
>lcl|NZ_CP026520.1_prot_WP_084706587.1_1143 [locus_tag=PC41400_RS06015] [protein=helix-turn-helix transcriptional regulator] [protein_id=WP_084706587.1] [location=1330686..1330898] [gbkey=CDS]
MAYTRGSCRLRLLLQRAGMRQAELARRTGYSRHQISNWVNDREKMSFDAAATVAFTLDCHMEELYEFHEG
>lcl|NZ_CP026520.1_prot_WP_042229728.1_1144 [locus_tag=PC41400_RS06020] [protein=hypothetical protein] [protein_id=WP_042229728.1] [location=1331099..1331335] [gbkey=CDS]
MEKAFQVCVLLFNQANEYQLTAKLYDSLGYKGQAKRYAHKAEAFNESAYIVRSCLGISFSDVIEAVSAAEACPENKEN
>lcl|NZ_CP026520.1_prot_WP_042229726.1_1145 [locus_tag=PC41400_RS06025] [protein=glycosyl hydrolase family 8] [protein_id=WP_042229726.1] [location=1331666..1333012] [gbkey=CDS]
MKKTLLGFTASACLLCSLVAAPVMAGAQGVPAASASEDTLVQSQAVNAVAAAALKPFPQHVSYTSGTIKPNNYTQAQLDQ
QVKTKYDEWKSRYLVKHPKISNQYYVFYNKEKIVTKPKDVVSCSEGHGYGMLITAYMAGHDADAQTYFDGLYRFYKAHPS
SIDPALMAWQQAKDSSGNIVDTSGSDAATDGDMDIAYALLLADKQWGSSGTVNYKSEAVKIINAIMKREVHAKYFHLKMG
DWASDSDPEFGPGTRASDFMLNHLKAFKAATGNANWDKVTDTTYNVINTVFKNNSPNTGLLPDFIVRKSDGTYAPAPEGY
LEEATDNDYSWNSCRTPWRIATDYLISGDTRALSQVRKMNSWIQSATGGNPAKIASGYELSGKVIEADHAIAFTAPFAVS
AMVDSSNQTWLNKVWSDIVNSPTSDSYYYDNSIRVLSMIMVSGNWWTP
>lcl|NZ_CP026520.1_prot_WP_042229725.1_1146 [locus_tag=PC41400_RS06030] [protein=DUF3221 domain-containing protein] [protein_id=WP_042229725.1] [location=1333158..1333673] [gbkey=CDS]
MIRNILLVIFVLVCLPGCIESPESLESPEQSKSTFTGYVAEKDTKANRALLVSNSESSLMGNEKIYDADWIAGIADQVNV
GEKVTVEITGMIMTSYPGQTSGTFLSKEKTEKPEGAVLEPEEVLRRAFHQEKTHIPTVKKISLDNEKNVWNVTLYDNSLH
KDIEVVIEDRE
>lcl|NZ_CP026520.1_prot_PC41400_RS06040_1147 [locus_tag=PC41400_RS06040] [protein=excalibur calcium-binding domain-containing protein] [pseudo=true] [partial=5'] [location=<1333858..1333965] [gbkey=CDS]
KNKGGKTKYKPYASKELYDANKKLDRDKDQIACER
>lcl|NZ_CP026520.1_prot_WP_042229722.1_1148 [locus_tag=PC41400_RS06045] [protein=hypothetical protein] [protein_id=WP_042229722.1] [location=1334594..1335283] [gbkey=CDS]
MKKISIFMLSVSLVLMFVFSSSGFAAPIDNKVKFEQYLYNLDKSDPLVSEQIQKYNKLSETEKSRLVELIYDPNVTKQAF
DSMKTVALGRSTSISNGEVVIQSKVEESSNKKRAITTASEESYWGSYSWTMTFLGIDLTTLKTKVFYTIDGNNNVLSADT
AVNSHWNIDPLVIVSDGVSSAYKSGGYAYGNGPFTVYGTATLGFISASVSVIVKSKPGDTTGWKESAIT
>lcl|NZ_CP026520.1_prot_WP_042229720.1_1149 [locus_tag=PC41400_RS06050] [protein=hypothetical protein] [protein_id=WP_042229720.1] [location=1335722..1336081] [gbkey=CDS]
MKVEMDLYNDWIETVKEIFRGSGAPLPQGWSAEEVGREYYLQTSPSEEEAEERRKANEQRLTDMQRTLMDNMETVVVPDI
REKTGYGGSHFRFRWIYSQGEHIVEECSQYRIPLGPSPE
>lcl|NZ_CP026520.1_prot_WP_042229719.1_1150 [locus_tag=PC41400_RS06055] [protein=MarR family transcriptional regulator] [protein_id=WP_042229719.1] [location=1336222..1336686] [gbkey=CDS]
MKEQDSTLVNEDIDLSLKLFVVLSKAQKVIMDRSVKDMKRHGLSPSEFAILELLYHKGKFPLQQIGDKILVTSGSITYNI
DKLEKKELLRRISCPSDRRVTYAEITERGTELFNDIFPEHAEVVKASMNGLTAEEKKLAIGLIKKLGTAAQKQS
>lcl|NZ_CP026520.1_prot_WP_042229717.1_1151 [locus_tag=PC41400_RS06060] [protein=pirin family protein] [protein_id=WP_042229717.1] [location=1336773..1337504] [gbkey=CDS]
MIKVHGALARITRNFGWLYTSHSFSFGGYFDPGNTEFGPLRVFNDDRVQPLKGFDFHKHEEMEIVTVVLRGELQHEDTLG
NKTVTRAGEVQRMTAGTGIEHSETNPSASEEVELLQLWFTPNEEGLTPSYEQAAYDLDSRKNEWVPVVTGRPGESAAGDG
DDGVKPARIHQDLTLYLAELEPGQALTFSQKPGRRMYVFVIEGSATADGAYKLSRRDAARITEQERLELHTETGARLMLI
DLP
>lcl|NZ_CP026520.1_prot_WP_042229716.1_1152 [locus_tag=PC41400_RS06065] [protein=nitroreductase family protein] [protein_id=WP_042229716.1] [location=1337636..1338262] [gbkey=CDS]
MKEFETVLKERRSATKFVQGVELPDQELDAIFSLVKYAPSAFNLQHAHYVVVKDPAVKERIYEAANKQYKVKMASAAILV
LGHMDAHHDAGKINEGLLNLGVINKQEYDQTIEAVTAFYEDRGEAFKRDEAIRNASLSAMTFMLAAKDRGWDTCPMIGFD
ADAMREALNIPDRYVPALLITIGKEDISGQRPRGYRKPVGEFVSRDSF
>lcl|NZ_CP026520.1_prot_WP_042229714.1_1153 [locus_tag=PC41400_RS06070] [protein=ring-cleaving dioxygenase] [protein_id=WP_042229714.1] [location=1338327..1339277] [gbkey=CDS]
MTKKTAGIHHITAFVKDAQANVDFYAGVLGLRLVKKTINFDAPEVYHLYFGNESGSPGTAITFFPWATSRKGRIGGGQVG
IITFAVPQGALPFWKERLRKLDVSFRETERFSETYIQFQDIDGLELELVERAEGADSKWSFGGVPADKAIKGFGGAVLYS
LNPESTMHTVEHVMGLTRTAEEDGLVRFVSEGDIGNVVDIKKDEVPHGSGGSGTVHHIAWRAKDFEDHEDWRDHVQTNGF
EPTPVIDRQYFNAIYFREAGGILFEIATDPPGFTRDEPFEALGEELVLPEWYEPHRATIQKGLPPIEVRVLEADKQ
>lcl|NZ_CP026520.1_prot_WP_174446812.1_1154 [locus_tag=PC41400_RS06075] [protein=alpha/beta hydrolase] [protein_id=WP_174446812.1] [location=1339280..1339891] [gbkey=CDS]
MKHVYREGSNPAAPVLVLFHGTGGTEHDLLPLAERISPESSVLSVRGNVLENGMPRFFRRLAEGIFDEEDLVFRTEELGG
FLDEAAEKYGFDRSNLVAVGYSNGANIAGSLLFHYKQALKGAILHHPMVPRRGIELPDSAGLPVFIGAGKNDPLCTPGET
EELKSLLEAAGAEVDIHWENYGHQLTGSEVDAAARWFADRFKG
>lcl|NZ_CP026520.1_prot_WP_042229957.1_1155 [locus_tag=PC41400_RS06080] [protein=flavin reductase family protein] [protein_id=WP_042229957.1] [location=1340026..1340637] [gbkey=CDS]
MIEIDPESQSERETYKLLIGSIIPRPVAFVTTLSKEGIVNAAPFSYFNIVAANPPLISVSVQRKNGEPKDTARNARDLGA
FVVHISDESYIDAINATAAGLPPDESEIELAGLTLVESGKIAVPGVAEAKIRMECVLERSIPLGGIESAPACDLLIGRVV
RFHIADELYSGGRIDPAALRPVSRLAGSSYSKLGEIFSLERPE
>lcl|NZ_CP026520.1_prot_WP_042229710.1_1156 [locus_tag=PC41400_RS06085] [protein=ATP-binding cassette domain-containing protein] [protein_id=WP_042229710.1] [location=1341451..1342197] [gbkey=CDS]
MGTRLILHGISKKRHDRPSDYLFRDISAEVGEGERVAVIGTSGQGKTTLLRILARLDRPDEGELRLHGKSADGWKPQEWR
KKICYVAQQPVMLPGSVEENLQTVSRLHGTPFDRGLAERSMAALGLEGLDWSKQGGDLSGGEKQRTALVRAMLLQPDVML
LDEVTASLDPGSKRAAERLLVDWSQSGGASLVWITHDLEQAAQTSDSVWFMDEGRLMESALTAEFFRGPSTEQARRFISR
PDREREEY
>lcl|NZ_CP026520.1_prot_WP_042229708.1_1157 [gene=fetB] [locus_tag=PC41400_RS06090] [protein=iron export ABC transporter permease subunit FetB] [protein_id=WP_042229708.1] [location=1342197..1342961] [gbkey=CDS]
MSNLALASTLVFVAVTILLSMRQKLGLEKEIIVGTIRSAVQLLFVGYVLHMVFDAKHPLFMVLIVAAMIAVATWNVGKRA
KEIPGIKTRIALALACTEAVTMALLLGFGIIEAIPQFIIPISGITIGSSMIVAGLFLNAMKREMEAARGEIEALLALGAT
AKQAIHASIKRSVRSGMIPTLDGMKTTGLVQLPGMMTGMIAAGADPVEAVRYQILIMFVLSSAAALTAMLLGLLSAPLWF
TEDGRLRVQDAKKL
>lcl|NZ_CP026520.1_prot_WP_042229706.1_1158 [locus_tag=PC41400_RS06095] [protein=Gfo/Idh/MocA family oxidoreductase] [protein_id=WP_042229706.1] [location=1343535..1344530] [gbkey=CDS]
MIRFAVVGTNWITEELIHAGRESKDFRLTGVYSRTAERARQFADKFDIQHIYTDLEAMAQSGEFDAVYIASPNSFHAEQA
IALMKGGKHVLCEKPIASNTRELSAMIEAAKENKVLLMEALKSTLLPNFKAIRDNLHKIGPVRRYFGSYCQYSSRYDAYK
QGTVLNAFNPVFSNGSMMDLGVYCVHPAVQLFGKPDAVKAGGMLLESGVDGEGSVLLKYGEMDAVIMHSKITHSYQSSEI
QGENGTMVIDKISQPEKVEIRYRDGSVEDLTQPQIARNMYYEVQEFIDLIRDGKTESAVNSHEVSLTAMEIMDEARRQIG
LVFPADKQTSV
>lcl|NZ_CP026520.1_prot_WP_042229704.1_1159 [locus_tag=PC41400_RS06100] [protein=sugar ABC transporter permease] [protein_id=WP_042229704.1] [location=1344855..1345802] [gbkey=CDS]
MKQDSSNVSRAAASAPKKPFWNHRRKEATIGWLFLSPELIGIVLLYLFPVLFSLYMSLCDWNLLGGMSSVKFVGLQNYVE
MFQDDKVLAAIKNNLIYTVLTVPVGMALSLVLAVLIHTKVYAQSYFKVAFFIPYISSIIAVGAVWSALYHPSLGPINGFL
KSMGIDNPPMWLADPKYALLSIAIIAIWAGLGYQIIIYMAGLSGIPDDMYEAASIDGASSLQKFYKITMPMLGPTNTFLF
ITLLMGSFKVFDLIAFLTSGGPNNASTVLVYRIYEEGFRNFRMGYASSISWLLFAIVGIITLISWQIQKRQVHYG
>lcl|NZ_CP026520.1_prot_WP_042229702.1_1160 [locus_tag=PC41400_RS06105] [protein=carbohydrate ABC transporter permease] [protein_id=WP_042229702.1] [location=1345815..1346651] [gbkey=CDS]
MSLRKFDFWKLFNTVIIGLFSLCFLIPLLWMTSAAMKYEKDVMVFPIEWIPKAWNAVGNFKAVWAGNVPFHLFYFNSIKL
SVIVTALTLIISSMAAFSFTKLNFRGRNIAFTLLLTFMIVPEQATLIPRFLLIRWLGLYDTHAALILMMMFSIYFTFLLR
QFMMSIHSDFIEAAKIDGAGYFRIYWQIMLPLCKPILATVAIIKFIWTWNDYQMPLIFLLDKDLYPITLGIQLFKDDYAD
NFAVLMMASLSAIIPLLVIFIALQKQVIKGIALGGVKG
>lcl|NZ_CP026520.1_prot_WP_042229701.1_1161 [locus_tag=PC41400_RS06110] [protein=extracellular solute-binding protein] [protein_id=WP_042229701.1] [location=1346791..1348125] [gbkey=CDS]
MKKATAWVLICVMLAITVLSGCGKSEESSAGKDSSGKQVTIKYYNWDNETQAQSTKKYIDQFQEKYPNIKVQHISLVPGN
SLESLKKLDVLLTSGEKVDVVMFPSISELDTRVANKVLAPLDEFYAKDNVKPEEEYFVNPKFEGKYYGIQNNITTNFVLL
NKDALDEAGLPVPKLGWTWDEFREYAKKLNKGEGTEKRYGAYFHNWPMYANPDAQIYMQHPFLTEEGKTNFDSPTYTYFF
NLRRAMEKEDKTAKPYADVIGAKLNYRAEFFNQKAAMLLSGSWMISEIGDTQKFPHTFKTAVAPVPVQKKGDSTDMFMGG
NYMSVGASSEHKQEAYQFMRFMSTNLTDARAELPGWKKGDAKPLAERLIGQNKDLYDVDSLMYTLFGAGIKALPPSKNSI
SYAAALDKILTDGFSMFILDNKKVEDVQKWMVDEANKVIEKNKK
>lcl|NZ_CP026520.1_prot_WP_042229700.1_1162 [locus_tag=PC41400_RS06115] [protein=histidine kinase] [protein_id=WP_042229700.1] [location=1348261..1350000] [gbkey=CDS]
MLQSSFYRKIQFSFLIFILLPIVTVSSLAYITIKRTTMDKMEAANQSILQLMTKDIEKLLDDISFASHFYVQNEEIRDNL
REIQGMTGIHTFDDFNKYTKIRNSYDQVSINALNKYVRLFLVNDTGFIVPYMDVGSGQTLPQMENDWRTVKPRIVLDNPT
SIQWLGTVAGSGPDKDTYYYLSRVIRDTDTGGLLGTLMIGISSTYFDPYFSSSTSGQFTLFEKNGQRIAGNASIAFTEPE
LPQKQGDRNIRSEAIIAQGNWKLVYETPEKTATGQITKTFFVSAMALAVFIIMFLMLSFFLARSLHRPIRKLQRIAHQFG
EGNRSIRFPDKGRDEISDLGRSFNSMLDEINGLIAGIETEQEQKRVLELQALFAQIRPHFLLNTLNSIKCSLYVSEDKLH
GRMIDSLSSLLRAYMRVNEPSTLRSESALLKHYLEIMNMRNDLALELSVRIPAELERQPMPMLLLQPIVENAVVHGFVDK
EEGGLLELTAERAEDRVVIRISDNGRGMTDGQLRELQAVLNQDPDQQSSYRRVGLLNVLQRMRLTYGMSAEMGMERSENG
GVTVRLSFPIQPLEGESHV
>lcl|NZ_CP026520.1_prot_WP_042229699.1_1163 [locus_tag=PC41400_RS06120] [protein=helix-turn-helix domain-containing protein] [protein_id=WP_042229699.1] [location=1349993..1351585] [gbkey=CDS]
MYNVMLIDDDVPMLKYLHKLIDWEACGLHVAGETYSSVRALQMFKELLPDIVVTDIGLPQMNGLELAAQFSSIKPGVRII
FLTCHEDFHYAKQALLLNADDYLIKDELTEDQLLRSLKQSVGRIQQSVVSLEQVSYREDVNRNRDLLKESLFQRLIKGTE
ADGLLGYAERLGVEWKHPAYALLLGHLSLAGLPEKYASRDMPLLRYGVYNIACEVAQTYEGVTVFNTGDFLVILLNYRPG
LTFNEKQYLHDYTADLRDKCDRFLHVGLRFWSSPARLELSSAGPTFRSMAKNRYRFYYADSGYAAPKFDDGSEVYYLPAE
GLFGQELEDIGAAAAHRDHALLEEAIERIRRTACGKLADPEELNRLCVRKVRQLFIQQARPGADAEEDEWIPYLQTAMHV
DDALAVFGWMIQRLMIGTDTGAAPAVKEPKLQVIDDYIYRHLGENVSSIDVAEHLFMNPSYFSRYFKRLTGESFTDYAHR
YKVKIACKALQMSEDPVELIAAKLGYSDRTYFSKVFKKYTGRTPREYRER
>lcl|NZ_CP026520.1_prot_WP_174446811.1_1164 [locus_tag=PC41400_RS06125] [protein=glycoside hydrolase family 88 protein] [protein_id=WP_174446811.1] [location=1351859..1353001] [gbkey=CDS]
MWKAAIEDAVLKTRRNIEKFGDKFPHVSDGGRYELRDNDEWTCGFWPGMLWLCYEYTKDEVFLEAARSTVRGMKKRMEEN
KVLAHHDIGFLYSLSSKAEWIVLGDEDAKNLTLEAADKLLERWNDKGQFIQAWGPMGDETEGGRIIIDCLLNLPLLYWAY
EHTGDEKYRKTAVIQADKSRRFLVRGDDSSYHTFFFEPVTGEPIGGATHQGFTNGSTWSRGQAWGVYGFALSYRYTGNAA
YLETSKRMARYFASQLPEDSVAYWDFYAEVTPETPRDSSASAIVAAGLHELIGLLEPDDPDRAEFADLLERSLASLVKNY
STAGEPEAEGLLNHGSYHVRGNLGLDDYMVWGDYYYLEALMRVERDIPGYWYDRKERTLG
>lcl|NZ_CP026520.1_prot_WP_042229696.1_1165 [locus_tag=PC41400_RS06130] [protein=alginate lyase family protein] [protein_id=WP_042229696.1] [location=1352998..1355349] [gbkey=CDS]
MKSGRIGEGLNAGQPDLATAAAAAAALLKPGGERRNEAYDTLKRHYETRTSPVLFIREEERERLVSHVRNHYGEDMREVL
ASAERVAAKTFLFRGEWDMEPTCVPVTFAEDIDWQHVPAGDVEWAYMLNRHAYWVDLGQAYLVTGDGKYAEAFCMQLDDW
LRRNPVPAGPAEASATLAWRTIEAGLRCINWIRAYECFRDYPGFTAELVGRMLKSLHEHGEYIAGAFDPWRRLSNWGVLE
NAGLFALGVFMPEFRNAAFWLEVSADRLRQTAALQIGRDGMHWEQSPLYHNEVFFQYMHVLHLAENNGIGLYPEWISAVR
RMAKANLYSVKPSGRQLMKGDSDDNDMRDALTAGAIQFGDAELKYAGHARVDLANLWLYGLHKAEAYERLTPVRPETASL
AFEEAGQYVMRSDWEPDARFASMNCGPMGGGHGHADMLHVDLCAYGRDLLMDPGRYTYYGGDEFRRYFKLGAAHNTTLVD
GKEFTEYLGTWDYGRIAAPLGTRWTSGPAADYAEGSHDGYLHLADPVIPKRRLIFVKPDYWLLADSFRCGDAHRFEQRFH
FAPGQVRIEERGVCFTTEDGTAGLHVLPVTRTPLLTETEPGTDGNSADASAGKTAYEQSGECADQAGVTASPASLSVPTP
GEGAVPADTGAAGRGPLHVRTEDAYVSYRYNEMTPAPAVTFSCGSTGPASMMQVLYPVRPGETEVPVVEQADVFTYNGAK
VPDSAAEACRLTWTSSGEQHIVVVCHQKPSTSIMSYVVEGVQIFGEVVLIKKTPEGIETINVR
>lcl|NZ_CP026520.1_prot_WP_042229695.1_1166 [locus_tag=PC41400_RS06135] [protein=phosphodiester glycosidase family protein] [protein_id=WP_042229695.1] [location=1355700..1361810] [gbkey=CDS]
MAKNRRRLASFLSGMMLCSILLPAGQGLAAVPGTTASAGQIVDKRVVEIGPGATYQWMNMKLDRGPEKLHFVEFDPKNAA
LELQAGKTGGKVYGMQGVSQMAADADRTGNRVIAAVNGDFYDMSTGIPSGLFMGEGEILASPPADWYAFGIKADGTSLYG
RSPKLTRTLAFGGQSSPLTEINRLRGTESLVLYTPEFHTSTMTNDLGDEVVLDVTGGAVQSGQTLAMKVSEVHKNKGNTP
LAPNQVVLSASGKFRDTLAPLKPGDDVTASFTLEGEWKDVRVAIGGSGLLVKDGAAQPNGDPAAHPRTAVGTKADGSVIL
LEIDGRQPGFSEGVDLTELGLLMKDLGAVNALNLDGGGSSTFVARMPGEKTRKILNTPSDGGERKTANGLLLVNKAPEGA
ADKLVVQPQTERVLTGSSAEFKAAAVDANLHPASFSGTPQWSVDPAVGTFATDGTFTAGSKAGETVITAASGGLTGTGRV
EVVEDLTELKFADAVKTFTSGSSETLKITALRDGQVVQADNSRIQWRTEGPIGSIDAAGVFTATAETEKTGKIYATYKNV
ETSLDVHVGVPPVILEDFENGLDRYNPSAGAQFKSTNVSIETDEEFVRFGKNSLKLAYDFTGMPGTSGAYLQVKSGAQPI
QIPGYPEKISMWVYGDGNKHWLRAQMRDGKGAVALDFVDQTVGVDFKGWKYLEAVVPKGRTLPLTMDMPVRYMETSSAKK
DAGAIYVDQIRALYGPAQDDMDPPVLKNFSPADGETIKSNLPTIKVYGEDYGYDPAKHPGTTLIDPDKIRFYLDGTLVQH
TLYPPEGRIYYTPGTPLADGIHQAKVSIRDLSGNQTTKEWTFNVDTGSAKIVYDTPKEVYAGGTYSLDVKGIKAASIKNG
RIDFAFDPGKVENIRLVRGPKLSDMQLTADIDGAAGLVSLKMKELQTASLTDQDIIGQIQYDIRKDAAGTNLIAFKSGSV
SFANTGDTSFSFFGLPLSSEIKNQLKLEWNDYGHVQGYETELVVKDERGAAVEGAGVLADGSSIGTTDAQGKLRTQALTA
AVKSYKLQAAKGNAYSAVSVFAVSPLGGNETPYNIGVNMGSDPASSRAFIWHTNPGTEPTVVELVKQSEFTDFQGGSVKR
ITGDSSLYSTLDIGTIRIHKAVADGLEPGTSYVYRVGDGKGHTSKQGTFKTAELAGDRTKFLYFADSQAGDQAGFNLWGN
TVKKAVADNPDAEFMVHAGDMVDKGFNEKEWNMWFGAAQDAFLNTTLVGTIGNHEVMGTRENSDFLAHFNQPGTGLDSLK
GSSFSYDYKDVHFVVLNSEYQYEEQKEWLKRDLAKTNKKWKVAIFHRGPYGSIYDTAELRSLWAPVLEENKVDLVLNGHD
HIYLRTYPMKNNQIAADGQGTTYVVAGSSGPKFYPVTKRDWQRVIDEEQTQMYASIDIAGDELKFVTKTVGGRVVDEFSL
YKKEVPPVSVEIEQQNLKLAVGESVKLTAAVKPADATDRSVTWSVYSSTPADAVTVTADGTVKAAGLGSAVIRATSGTAP
DVYGETTVTVDRLPSGQLASIELSGKGSLKVGETDQTVTRAVYTDNSRFDLTDGVTYSSSRPSVAQIDTHGAVKALTEGT
TVLSAVYGGFTAQYELTVKASGSGGSDGGSGGGSGGDNGGGTPPVPPVPPVEPPVKPEPGSAGKLAATAAELAAMKAAGA
VTFVTSAPFEELALPGNAGALLDGTAVRIQAGNVAVTLPAAVLSELGELVPAAEREGSMLKLSAARIAGEDARRLLGSAS
AASGARLAAGSDLLELRLTVTTKDGKTSELTRFAKPVTIELPVGSGVDAKLAGMYQIAGSGALEYAGGTWHLSKLTADVN
HLGTYAALEYGKAYADVPADHWAAGVIGELTARHLIDGMTENTFAPERKVTRAEFTAMLVRLLNLKAEGVPSFSDVSGQD
WYAGAVAAASKAGLVNGVNASGFAPEAVIKRQEMAAMIVRAYEAAAGKKAAVGEPVRFADVEASPAWVREAVDAASSLGL
VNGRTAGGFEPEGTATRAESAQVTANLLKMLGGVTR
>lcl|NZ_CP026520.1_prot_WP_042229956.1_1167 [locus_tag=PC41400_RS06145] [protein=response regulator transcription factor] [protein_id=WP_042229956.1] [location=1362299..1363015] [gbkey=CDS]
MQNTLLLVEDDTEISDMVSDYLTKEGFVVVRAYDGEEAVRKFAGRQFDLILLDLMLPKLSGMDFLQKVRQTSLVPVLITS
AKDGDVDKALGLGFGADDYIAKPFSMIELAARIKAAIRRATQYAAAAPAERTPAPPEVLRVGDLVIDLENFSVSKNGAEL
KLTSKEFHLLKLFAVHPKRVFTKAQIYSSVWEDDYYGDENVINVHMRRLREKIEDDPSNPVYIKTLWGIGYKLGEFGE
>lcl|NZ_CP026520.1_prot_WP_042229694.1_1168 [locus_tag=PC41400_RS06150] [protein=sensor histidine kinase] [protein_id=WP_042229694.1] [location=1363012..1363968] [gbkey=CDS]
MSGWTGLLLALLAVSLVLHFVQYRSKKKRDESLAYITARLDGILSGGAARGKLLHLTDDKQLIALLNRLNGLVETNQRMT
ATYNKTEDSMRKMLANISHDLKTPLTVVLGYMETLIQNPSTPAEERERLLDKVHAKAREVLVLINQFFDLARLESGDRDI
PLNRILLNEVCSQTILAFYEPLTSKGVEVVIAMPETPIYVMGNEEALVRILNNLISNASQYGSDGNMLGLTLREDEEHAY
VEVWDRGKGISEFHQDRVFERMYTLEDSRNPLYQGSGLGLTITKRLVEKLGGVITLTSKPYVKTSFTVKFKKTAYGRT
>lcl|NZ_CP026520.1_prot_WP_042229693.1_1169 [locus_tag=PC41400_RS06155] [protein=ATP-binding cassette domain-containing protein] [protein_id=WP_042229693.1] [location=1364057..1364980] [gbkey=CDS]
MGTLLKTTRLTKVFDGKEVISDVDMTIRQGEIYGFLGLNGAGKTTVMKMMTNLIKPTGGEIEIFGEKLTENSYEILKRMG
SIIEYPVFYDRLTALENLELHCEYMGFHNKAAIGEALDLVQLRNIDRKPVKDFSLGMRQRLGIARAITTRPELLLLDEPI
NGLDPVGIKELRSLFKMLSKEYGITLLISSHILGEIEQIADTVGVISEGKLIEQVPMDEVRGRNTEYIEVVTPDLNKAVY
VLESELGMRNYKVVGDTTLRIYERGVSQAEISKGLLLNGVQIEALNKKGSSLEDYFLQSIHGGGVHA
>lcl|NZ_CP026520.1_prot_WP_042229690.1_1170 [locus_tag=PC41400_RS06160] [protein=ABC transporter permease] [protein_id=WP_042229690.1] [location=1364973..1365668] [gbkey=CDS]
MLKLMKLELKKYKIGRYFPGILIANAVILLLISGIHVAEQVDGGIDVFQDYKMAFGVIIGFVKITFVIFASVILAKLVIG
EYNAKTVNILFTYPVRRKKIMAAKLLLVSLWTFATVFLSGLFVTGGFLLADASLNMVHETLTAPMIVDHVLRMLLNAVSA
AGLCLIPLLFGMMKKSVPATIVSAVILVTITDSNNMGFALSAGYILPLVFAAVGVLAAFWSIRKIETSDVL
>lcl|NZ_CP026520.1_prot_WP_042229689.1_1171 [locus_tag=PC41400_RS06165] [protein=glycosyltransferase family 39 protein] [protein_id=WP_042229689.1] [location=1365757..1367016] [gbkey=CDS]
MFTLKEENLRTKKILYALMAIVLAVSAFTALYYGDAFLLGTYEKMDNDDVKYMYAAKVLLEEGTLVYKSYPNPTLFIMPG
FPVMLAFFMWIFGDYENTAIAVRLFQCVLQAASVYLLFVIARHMFNSRVALIACIIAALYPPNYYTSGVILSEGIFQIIF
LLLVCMTLYALQKPSVGKYIAIGALLALCAYFKPQFILFPLVIGVTWLIRRVPFKKMVLYTATIVLTACLLLSPWWVRNY
AQFGKFIPFTDSSGNPMLMGVLPGEPSPGFFKEYPQYSPDVYDDSTEKVGHIILYSLKTKPFETIKWYLFDKTILAFEQT
FYWLDFWGMSYHFVLALHIILCVLGVAGFIAALINRRGDILVAALTVAYVIAVHIPFVQFSRYMYPIIFIFTIFAAYLIA
VPFGSLSFRRSSSVTRHFP
>lcl|NZ_CP026520.1_prot_WP_042229687.1_1172 [locus_tag=PC41400_RS06170] [protein=S-layer homology domain-containing protein] [protein_id=WP_042229687.1] [location=1367228..1368226] [gbkey=CDS]
MKKGARAATAVLLAFSLSGVYASAIYGEGSAFKDVPGDFWGRDVIEWAVGNKLVDGYPDGTFKPAQAVVQNEFLAMLIRA
YQPGDFKSAPDSSDWASPYVDYASRLGWPVVQTSAIDRNASARKALTRGRAAQLLASASGRHYSVDDSIRYLLDVGLAEG
KEERTPEGFGKNDTLSRAEAVTFVQRLRANESRLRPSPGSLEVYDLSKLNTVAYANKTYGYSFKLPAGWAGKIDIEEDGD
SVRFIHRASKAAGGLLFSINVGSKEDYEQTSELIRTVYLGAKGDRYVYICYPSDVQYSLEDAKSSAEYHDLFDGIKSLKP
TFQWRDSALSAQ
>lcl|NZ_CP026520.1_prot_WP_241688798.1_1173 [locus_tag=PC41400_RS06175] [protein=hypothetical protein] [protein_id=WP_241688798.1] [location=complement(1368360..1370135)] [gbkey=CDS]
MLGSTYHRSIRYRNGAILLLVLMLGLAGYKGYSVWHKADLYARAAAEQAAGRDVEAENLYMLARDISSFDYKNNEISAAL
SALRPVTELKRLFASLASEIAVTNGGSQEQARLLQAYAAYGAKKAEVAAADESAKKRFGELTDAYKLDEQFAAALAGAKK
SLTALLQQGNGASAQEGESAMSLLLQLPASVYKDDKTKRKDLNTLFQKYDTTRLQTIFNQKSFDEALGEIVRIRQFYSER
KVEAAWVLPKAETYVQGTLTRLLETNDVPGFLTHAKTFQNLKELSGSSSKVNAFIQSSLRKQTAKAEQLASAGKYADAIA
LYKVLGDYKDTGKEIRQTEIRWMAAEPGVLLAKAKDNVRFVQTAGAKGLWGSEAIAAGVTDRGTLVTARLLADGSVVSTE
GEIGGGLAVKAVSFADGLGAGKLPVVMLESQSRTRNARYTAYEVDASGLNALLDVEADGYKLQSAGVLLVRNPAGAGSGQ
EAVYKLQGGQYALAEVPQRVPDPSLEAKDIQPSELPQQQAETVVRLPLAITRIGSGMAVAEMDGLNVLLSGKFSWKPGNA
VVTGTYTGETKILQGDRSVQAYKINVNHIEQ
>lcl|NZ_CP026520.1_prot_WP_042229683.1_1174 [locus_tag=PC41400_RS06180] [protein=hypothetical protein] [protein_id=WP_042229683.1] [location=complement(1370150..1371088)] [gbkey=CDS]
MSGKIPVELRKARQTVHTYERLKQCTRCGTYSVLWHDNCLSCGAERKFLSIGDIVSSVLRRNIQRDALILAAIACAAFYL
ARTFQEMALSAGAGVLLIAVYAFLRRRYAEALMNNALRKLLLGENRAIRRGILLDVDDAASDMEDGDFKSAYEKLREIGA
LITGSHIRNLKVYCLNRFILRSDMELELGTLIPDEFEPGFITYLYEAAKVNPGLVQRDTLDYVQKYRVQIGGMVEGRATL
ALVAAAALRVKEYAFEYASLLYDFVDELPRERFLRLCRLVKDNPDRAPELRQKVLETAARKYGNDPEFEGFL
>lcl|NZ_CP026520.1_prot_WP_164977109.1_1175 [locus_tag=PC41400_RS29660] [protein=hypothetical protein] [protein_id=WP_164977109.1] [location=complement(1371085..1371231)] [gbkey=CDS]
MLTMIVLALLVLFVLIRWLTWNSAKRAALTSESIDREFMTELNEGDPS
>lcl|NZ_CP026520.1_prot_WP_042229682.1_1176 [locus_tag=PC41400_RS06185] [protein=VWA domain-containing protein] [protein_id=WP_042229682.1] [location=1371446..1373200] [gbkey=CDS]
MKLQRYAYMALALLLLVMAVPGGVFAADAATDTKIDAVLAADVSTSMNESDVNKVSYEAMKMFVDMASVQGDKIGVVAYT
DQILREKALLKINNAGDKKDLKTFIDQLTRGPYTDVAVGLSEAVKVLESGAEQGHVPLIVLLTDGNNSLNPGKTQADSDK
VLGEAVQKAKDKGIPVYTIGLNADGKLNKDKLESVSKETGGKLFVTSSAEDLPKILSEIYASHLKLKVVPLNGFTANGQF
QDVKVTIPNGSVLEANISIMSGKPVEAKLLDPKGKEQNIPSDKIVYTSSNAYSLIKILNPAQGDWTLKIKGVPQEKIDIN
LVYNYDVQVVLDAQLPKTAKAGDSIAIKAHLESGGQKAADAGLYKNAKASLVVTDLDTKQASEAPLTLTDKGIEGSWKVA
DSHAYEVKVKVESANFTRESAPVSVSAKSGGASSSGEASGDKGVNWVKIGSWIAGLVVLLAAALFGLAFWKKANKGFFGQ
LVIEIKDEETGERTNPQYKKLSGFKGKVQLHQLLSLAPEFAETKGIVFKPGPNDTLLLFNESACTIEKGGRVFDAAKGKE
LKKNDRIRVTVPNIQKSVSLEYLK
>lcl|NZ_CP026520.1_prot_WP_042229680.1_1177 [locus_tag=PC41400_RS06190] [protein=tubulin-like doman-containing protein] [protein_id=WP_042229680.1] [location=1373264..1376653] [gbkey=CDS]
MKATVREHIQQLDVSLGGGIISDKIRVDTIDNPMLVIGLGGTGIDALLRLKYQINRRFKLPEDPFTKKKKEKPNNVEFLA
IETNEHDRNKKYKGIGLDPLTEFVLLSNAEIGSVLQNRSILEPYIKEWLSPELTITDGINGASGVRQAGRLLLFTKIVQV
VQTLERKIKTLSIGTNKKLMVFILTGLSGGTGSGCFLDISYIVRGILERDYGSAGVDKISLLGYLFTPDVNLANKSLTEH
SREYIKKNGYAALKELDYWMNVDERTERFKQQYANILTVNSPMPPFNLAHLISATNLEGKLLENAYDYCMNVTAENITNF
MASEDKQSGEEFAIHDYISNIRTNINQMPKAYAANYQYNILGASSAVLPIEEMTTYLAYQVFKRMEKMFDAGPSQEEVEK
LAHKLGLDPDSLHREFDKRVPEPIPGYQNSERLSYTNVIKQQVVNIDTELEQSFLSRAREEYIKSRKQLPGQVKEAFSEQ
VTKIFRNPEQGPFYVSRIILQDKGFCLLKLISSYIESLKESLYRIPRDIQAAEENSEQKMTEARSAFISKDRKKDAYIEA
KIQEYLLHADRERMEQMIEFYEDLYNLINQENSRIYLVFTELLSELNKIFEKNGDILTRGDEEIDHKGNRTYYWNVVSVP
DIAKLIGQVTEDRNADELIRDFTNELLEKSNQWIKEQEVDVIGSVSEFLTDHFGELITKSMEEYLVIKYGHEDSIEKLVE
GRIASRLYEEAKPVFHLTNSSGHFNFPSWGFVSVPVQAPNIFKGIRNFQDNAIANSRFTIKESEVKNRIFWLNTQNGIPL
FLYTPLKVYEESYERTILEPEGIGRHLVQTDRVNWTYLPSPIPEKSWGETYENGRVKSYNASVRSLFERAAEYRSVRLKG
EDSGTINRYECVLTKPFDVEAFISAYNLQLGSQRPNFGELQRCVNDLKALLQTGLEADRSKDIFGSVNEELAKENLIRSP
ELLKQVREEVRKYDNIVRKIGELEQALSAHQDEEKFVAQYIEAMYTSTIAKKGALYVYDHDADEDPWEPFVNLMQTNKFV
EYEIYSKMRALDAKRLSLLQRKSSRRSQEMTASEDISGLVETLRGMAASYQEAKSALDYDRSEYAAGEDMYQFYKLVAGK
VNDMVKSLK
>lcl|NZ_CP026520.1_prot_WP_042229678.1_1178 [locus_tag=PC41400_RS06195] [protein=hypothetical protein] [protein_id=WP_042229678.1] [location=1376806..1379367] [gbkey=CDS]
MREKLVAFANEYAAETDSTGLDGDEQRSIHHPLVFLFLGDTSAEALQTMAEINNRRWNNSTGVVYLHVGTERSREESGGP
HAEHEAAARQDWPQADPPEAEKRSGQAVPDYRDLPRSGGTHRSDPAGGPEAGNIYTWTLPVPGNNARTARPDLHDSFRSN
REKLLELNVLLRRMNSRISEFGRLYSNLQRLNIAVVTRLDDPCNVLLPELTALMETIFGEHFRSVIVDLYGLLEEKQVGE
EFALRASMSFSFLRELEVLQSRDYTFSSPLQVTGEGITLPAQHGPSPLFDTVYLLSDKDERGIFTESGMKGSYETISSLN
LLKYRHIKAENEGDSSKAGTYNHQHFKQNIAPPGEEGVGYVSAGYSRVSRPNSAIALTVLYHLYRHVREKMEENVQGGAR
LLPDLISMEPQLWDREVESVVPGREQAVEEMYGLMHDPVSYGELKSMSLKEAELALFQDSARRFFDMNVRQRAEAAFAER
GFAKRLERTLKERVVDDPAYGYYALNRWLAEESGAGSRSLPGELRQQLKDAGNRADQLRDELDNFYGERVDRQPIGKGSL
FGWKSDKASVKNLIRHLLDEAYGLRRDLLVLELKQKLMTAYLDKLEELRERAAAAVRQMDSLEKLLLDTSRSGITAASDY
LGRNINEYYEHVVKQILTEMEGRRESRFYFDERFLGNVSVEIDRDAAQWLRKLMATARRDVLVHPLFHQTFEAELLERAN
VAASYDNRDVLSKEDLFRDLYTTLENEAAVRIDVYHSTHRNRHEEKYFFGDSRSEFIQYAFAVDGGSRTYKLGCVHEKKP
GGIEKLRLMGGFRLEDLMVYRNSRRYYETYTENGYRFHSMELPETGTDSTKLS
>lcl|NZ_CP026520.1_prot_WP_042229676.1_1179 [locus_tag=PC41400_RS06200] [protein=vWA domain-containing protein] [protein_id=WP_042229676.1] [location=1379427..1380710] [gbkey=CDS]
MKQRKWNPLLTLFSVIGGFVGFIVGEFILNRWEGSWHETVLMGVYFGQLALFVGLFCLVAEMISPQLNGKNWRLRYAGDG
WKFLVPATLVMLFIAGALFQVLYGFQVLGTKKAPQNYVLVIDKSDSMSQNDPQKQSLQAAKSLIGRMDGKNSAAVFTFNE
ETQLIAPLTKLTGSGREELRGRLDKIGEPTGQTDIGRALTTVMDQLKKDGAADSSSSVILISDGYSDVNTTEVLSPFVKA
GISVHTVGIDDSQVEGIQLLQRIAEQTSGSFHDVKSADRITDAFEQIYTAQQPWHLAGERLGAAASDGFHAFLRVAFILL
IGTLIGLSLGIIFDNRHLALSFAIGGTLAGLLAGLVLETGMPGTNIPFFYRAVADIVLAFVLSLGALLIPVSTNDSHSGL
RKNYQRARLEEGRYGRKDSAPVGKKFR
>lcl|NZ_CP026520.1_prot_WP_042229675.1_1180 [locus_tag=PC41400_RS06205] [protein=hypothetical protein] [protein_id=WP_042229675.1] [location=1380901..1381536] [gbkey=CDS]
MQWEEINEENRITKLASQPAREGECLLTWQWPKEVGAVYVYSFESGSEEPPEQLAPHQLKLFTREEYKASGGYRERMEYI
GMRGYRIFPCVMKDGRVAPYRQTGTDNTIRVSAGKARIRYSIKYGGRLFGKYKSAKIQLFCEIPVPQEALCYVKKEGGHP
AGRQDGTAYPFRRDFEAGTQTLPEIEIGKHDFLRLFFTDSRYGDVYELIQE
>lcl|NZ_CP026520.1_prot_WP_042229673.1_1181 [locus_tag=PC41400_RS06210] [protein=hypothetical protein] [protein_id=WP_042229673.1] [location=1381549..1382814] [gbkey=CDS]
MLGYLKSLFEKKPPEKERPPYYSIVCPYCFNKFEPNDVVFRATHIKDSDDDFMLQEDERLNSWRRKFNLSEVDMEAVILP
STIPDSYKTYVQNVLVAVTDRYGETTRRRLCPYCHNELPISAGKVPSNIISIVGASQVGKSVYMTSLIHTLQHTTASNFN
AACMPLSAEISRKFRQHYHEPIFERGSMLQSTNPNEQQEPFIFQFVFKDEREAPLTIVFFDVAGEGMVQREYLDIYASHI
KNSSGILFLVDPLQIRSIRDKIQINVGGEQGEFANRYDEPREVVISLFENFIAHQSNSKTEIPTAIVLTKSDMLQYLKEE
DSEYIQPNSNVFRNVVHQGYLDESEFENINGEIGRFIEKVDRPFKDAVDVYFSNTAYFAVSALGTNPVNKQISGVINPTR
VDEPFIWLLHKLGYIARRDAT
>lcl|NZ_CP026520.1_prot_WP_042229672.1_1182 [locus_tag=PC41400_RS06215] [protein=cadherin-like beta sandwich domain-containing protein] [protein_id=WP_042229672.1] [location=1382814..1385711] [gbkey=CDS]
MAKHIQQHLFTRDREGVFRTSEGFDTVAKSPGLDPSFIKSALHPYCVYKAPQELLERGEENSGLYPESVVAYHAENGDFV
IGRSVYVATDFTGQRSASFTHQYVIPKERKEAYIREPRRIFGIRGFQSRYDIQGGKEIPELEDIPFEPEIRAGEGDKLLE
KLEIDRDLFRQLVFAVLASVANKKKVFISLNTDISESAIYAKRLLELLYRCLPYAVRRQLGFMTFNSEPEAKQHLHVVFT
EKGSIRLPDRKIEKDFIFDFAGGRFLNTELPAQDHAYLDYVWSNRNNPAELQKYFDFCDEAVSGLDTQTLLSAAAYSRLC
VLYEIERGRTELYEQDRAAVMNGILSFFTTESYPRKPRLKELFVGLLTREASDKGRAVAADYAEAVMRYEKFAEPQLKAL
IARCLGIFVSRAADADAGDGIEGAAPLLDLLAAYPAVFAGVFGGLMASRPALGEGYIRYRMKRVKSVQAFTGEIGFWQSH
SEELMGEELFSREAYDPLRKLMRTDLRKKVALARTLTDYFEQLRQNSAGEKRHGVRQFSRWMKLAVHLELADDLKPSGLG
IEDIQGLGFLIDPINPDLKGQASRDQEGVLRVLQAAYKVLTLDGKKQSEAEDVMARLDPVDLDRAQLLLQKLMQNEVTPA
RFGQLVTAFHKPGTGRGPDRQPGYDYERLLNFVASAGQGTDTFTDFAAWSAEDIRFQDGKGAIHPYYKAALSRAFDVHAP
QAFRNKALSRKLLETGNPSFAALFETVRLRQSGKWARIWAKNKRKIVLRGSIFLILFIAIIAVLWKPVSDWMAPTPVVIV
GDLQEKTGSETVTVKASLKEPDKNTKLYVNDKVIENGSIDLKLNEGENPLVFKAVSKSGKASETVSKKVTLVPRPLVKAE
QPPLTVKAAKVTVTVKAEDGNDPSPEIYINGQLAGTGSVSQQVQLIPGDNPIEITAKNKLGRLSEPVRYTVKSNPVPVGP
PAGAR
>lcl|NZ_CP026520.1_prot_WP_042229671.1_1183 [locus_tag=PC41400_RS06220] [protein=TIR domain-containing protein] [protein_id=WP_042229671.1] [location=complement(1385885..1386319)] [gbkey=CDS]
MLPPAKPYKTSLYLSCAWNDRSAFRELVALLQTCRDRGFRFDTFLLPEDHPAAGADNERSLYEAVRGKMKYCQAVLIPAD
APSAYSRWINKEIIAATSEMNKPLIAVRPPDSGEPSLIVKQNADRIVGWDAEAIVQAILELSAP
>lcl|NZ_CP026520.1_prot_WP_241688799.1_1184 [locus_tag=PC41400_RS06225] [protein=glycine betaine ABC transporter substrate-binding protein] [protein_id=WP_241688799.1] [location=complement(1386347..1387195)] [gbkey=CDS]
MSVVLAVALVAGCSTGGGAKGKSVKLAYVAWDSEIASTNVAKYVLESKLGYKVEMLQVDAGPMWAGIADGSADAMVAAWL
PGTHASYLEKYKSSVEDLGANLKGTKTGLAVPAYMKIDSIEDLKKADVSASLNNQIIGIEPGAGIMMSTEKAVADYGLSG
YKLVESSSAAMAQELQKAYKEQKPIVVTGWTPHWMFAKMELKYLADPKGVYGGEEEIHTLARKGLKEDQPNAYKFLDQFE
WTPDDMAKVMVAIQEGKSPEDAAKEWADANADKVNAWIKGIE
>lcl|NZ_CP026520.1_prot_WP_042229669.1_1185 [locus_tag=PC41400_RS06230] [protein=proline/glycine betaine ABC transporter permease] [protein_id=WP_042229669.1] [location=complement(1387234..1388070)] [gbkey=CDS]
MNLPKIPLGRWIEFLESWLTDHFGPFFGAIKAVIAAMVGSIETALTALPALVLIVLITALAYWIGRWRIALFAFIGLLLI
DNLGFWELSMQSLALVLTASILSIVIGVPIGILCARHTLLQNIVTPVLDFMQTMPAFVYLLPAVSFFSLGVVPGVIASII
FAIPPTIRLTNLGIRQVPEELVEAADAFGSTPAQKLVKLQLPIAMPTIMAGVNQTIMLSLSMVVISSMIGAQGVGAYVYR
AVTQAKTGVGFEAGIAIVIMAILLDRLTQNVFKKKKSN
>lcl|NZ_CP026520.1_prot_WP_042229667.1_1186 [locus_tag=PC41400_RS06235] [protein=glycine betaine/L-proline ABC transporter ATP-binding protein] [protein_id=WP_042229667.1] [location=complement(1388067..1389266)] [gbkey=CDS]
MPIIEVKQLTKVFGSDPKRALTLLDQGRTKEQILKEHRLTVGVNKAEFSIEQGEIFVIMGLSGSGKSTLVRLLNRLIEPT
GGQVLFNGTDVLKMNPEQLRQFRRKNIGMVFQKFALFPHRTVLENAEYGLEIQGVERKKRTEMATVALKSVGLEGWENSL
PDQLSGGMQQRVGLARVLANDPEILLMDEAFSALDPLIRKDMQLELLELQSKLKKTIVFITHDLDEALRIGDRIALMKDG
SIVQIGTPEEILIQPANKYVERFVEDVDLSKVLTAAHVMKKPETISLERGPRVALQLMRDSGVSSLYLTSKGMKLLGIVT
AEDALQALKEGKSLEQVMQREVPTVSPGTLLNDLFEMMAETRLPVAVVDEEERLKGIVIKGAVLSALAGNAEPEAGGVR
>lcl|NZ_CP026520.1_prot_WP_042229666.1_1187 [locus_tag=PC41400_RS06240] [protein=transcriptional regulator] [protein_id=WP_042229666.1] [location=1389520..1390128] [gbkey=CDS]
MNDLSGFTTEQSQKIQKNRERVIDSIGKNMDLYGITLSIGHLYGYMYFNNGPVTLDELSKTMGMSKTSMSTGVRTLMDLK
MIDKVWGKGTRKDLYEVEPDWYQNFSDFFSIKWRKAVEGNMNALNKSLVELRHMKKEYEESPELQAVLDADEAKISHAMD
YYKWLLRLIEAMETGKIFELIPKEPREPASDEAGRPERDGKE
>lcl|NZ_CP026520.1_prot_WP_053228704.1_1188 [locus_tag=PC41400_RS06245] [protein=hypothetical protein] [protein_id=WP_053228704.1] [location=1390432..1391241] [gbkey=CDS]
MKAGLFKVMSLVLVFCIMLSQVANAATDTLKESLTSYQLPVRDLNGNVIGTQNTTLRLELSKEKNVETVTMFDETTYNFF
KDEDVYKKVYKNGSRVTKLEKQANGEFFINGSKVNLTINETKASGASPRAVDSGGKEWLTYYTNFSDNPDLYECLGYPSA
NNFLDYGTGTKLDRFVKPIAPYTMYKLNDFKMYANSVANARSNINTASVTLTAAIGTLVLTWETVIGAMVGTGGAAVAAY
QIYVASQDGKDAMASAYNVLKGWQPGVNT
>lcl|NZ_CP026520.1_prot_WP_042229665.1_1189 [locus_tag=PC41400_RS06250] [protein=hypothetical protein] [protein_id=WP_042229665.1] [location=1391874..1392245] [gbkey=CDS]
MTLNEKIAAYVELMNEAREISNRANSRAYELRQTAVRIGFEATAAAEHAALMAEIVRENTESGSLSKLIHASREAASRAA
EATLKALDAIKQANVADHEATEAFERTREAAEKLTEALQVEGY
>lcl|NZ_CP026520.1_prot_WP_241688800.1_1190 [locus_tag=PC41400_RS06255] [protein=helix-turn-helix domain-containing protein] [protein_id=WP_241688800.1] [location=complement(1392323..1394587)] [gbkey=CDS]
MMPFNKTRPTSLFLATLISFLVIIGLFAAFDLAAFAYFKTKYHGEIIAHNQSMLKNTAEKYSTHFSRIKTLLFNTYNDPG
VIAFNRHLQGATDEELRYLEAKSIVTRLRSDTSNPHLFLDNLEVYFLPHGVTLDKDGSSGAEDQYNRLYKSSAYPLFFWE
NRLKSADAFSLLPAADFKTVWNSPAEKKLLPVSFHLSGSSYLVTAYLDVDKTQRDFLGEGGDGRRFYVLNGDGSVLFGTG
EDESGAPVGLPAFEENSGTAVAGGMVYFRYTGDDGLTYITGVPNASVVSQLRKLNWTLLAIFLLSLAIALAVSLLSSRRI
NRPVKQMVSAVMSGSGGEPSAIREFAFIQEKIADLLKEKTEIRSELRSRQSLLTKYSYINKLRSIHTDIHEWQDFIASDE
SFQIIVYELRFRQAKPGEEPVPHERLASYYREHLDVLLREHSPGSHTFQMEKNQILTVISSAVAAGTLQAMLGDIRHLFD
SHEQQCLAAVAVSSRFEHASQFNDAYRQALERAGQATLAEETQVILETRSGLQACALSPAREQELNACLQAGDDKGALEL
IGRQLAEMRESGAGVVQYRQLAESMAGKVWKLMDICKIAQDAAWSVKPLIRRLKECCTFEDYIEGYRELFFSAAALIYEK
KDTEDPVIRFVMDALRDGYGEELTLENLADRLQMSSAYLSVYIKEKTGANFSDHLNAIRVSKAQELLTGTQLNIVEVGLR
IGYRNITSFNRMFKKWTGQTPGEYRRAAKLTPAE
>lcl|NZ_CP026520.1_prot_WP_042229664.1_1191 [locus_tag=PC41400_RS06260] [protein=hypothetical protein] [protein_id=WP_042229664.1] [location=complement(1394792..1395664)] [gbkey=CDS]
MNADRGRTVNSVYIDASADAIWWYVATAEGWKHFLSDIATFSGAQPEIAQGDELTLVIGELTNRAVCAECVKPELIRFDE
RYTSILPDGSLWEYNLRTSFAFEPIGCGTCTRVTVTVEGYTEDETMQWVRECGETGWRQSLFHLKNTIELGLDLRNSIFN
YPRLGVMNYTATAAQLEEQGLAGAGIRGNFIATVYPNSPASRAGLHGGVIVTSLGGMPVPTYHDFVKALSRFYGTNAPAD
IVFYERGERHAVTADLTYDDQFTGMIDPNEVSLDEISVKRRKLSAVKEQE
>lcl|NZ_CP026520.1_prot_WP_042229662.1_1192 [locus_tag=PC41400_RS06265] [protein=glycosyltransferase] [protein_id=WP_042229662.1] [location=1395851..1396750] [gbkey=CDS]
MTKMNVVIPVYNQAYALSLTLHGLTRQAAPYNRCPVIVVDDGSEEPIHAVAEAYSGRLNLTYVRIPRSGRAAARNAGARL
LDEGCVLFCDADRIPRPGFIEAHEQAWRKHGGHFVVGQVRELYVPDAEQNRDLVLARYGEQKYDRIPQYCRLVYRLFNES
GSTVSPVPWIAALSGNMSLPLDLFRDLGGFDDTFREWGFEHLELGYRAHRRQIPFAYCREAVNVHLAHPRAGASYADFIR
RSHDTFFRKHPHPVIGAFYDFMMGRISRIQLENLASTGMLGSAGRNTPDEYVRILNFKA
>lcl|NZ_CP026520.1_prot_WP_241688801.1_1193 [locus_tag=PC41400_RS06270] [protein=M2 family metallopeptidase] [protein_id=WP_241688801.1] [location=1396777..1398351] [gbkey=CDS]
MSQQRVQTKADSFLGKTNRELAGLYVPMMNSLWKVITTGEFRYVKQCEQDEKDYQEALASPVLRKQTEYWLNRPGLSADQ
ERELQVLKAEMAEYRMEADERQAAASLWNDLHYTIGTFRAEAGGALLSEHAARVLMRRLKDGPQRKQIWQAEMRLGGLVA
PKLLELVKLRNRTARRLGYADYFAMKLEIAGVGTDTLTELIRQIRGGLDFAYREVKCGIDLELADRFRIARSEVRSWHYT
HPFFQMHDAMEEPLGAGTDEMADKVSGWLLERGFDLSAVIRKADLAERPGKGQANCCLNLDRGRDIRMSCNLSPDTKGLA
VLLHETGHALFEKELDPSLPFLLRQPAHPFLSEGVALLLERLAWDAEWREGLFGGQPEGDARSGQERLRKETLVRLYWTL
TLVEFEKQLYANPDQPLNGLWWDLVEEVQGIPRPAEWDYPYWAAKAHLSTLPVYYYNYLLGEAAASQFQHTLQARFKVWH
GKEALGHLRKEVFRPGASASWRDLLAAGTSQELNPLFLIRQLQG
>lcl|NZ_CP026520.1_prot_WP_042229660.1_1194 [locus_tag=PC41400_RS06275] [protein=hypothetical protein] [protein_id=WP_042229660.1] [location=1398379..1398672] [gbkey=CDS]
MATIHIQKRTIVSRMERDSTEPVNSRVKTYTLTPEELDYYRRLPLSAADKAFKPGLKVMAMERRDGDGDGSREPDPAVGP
LVGGTEGDAHAREAGGA
>lcl|NZ_CP026520.1_prot_WP_129112456.1_1195 [locus_tag=PC41400_RS06280] [protein=hypothetical protein] [protein_id=WP_129112456.1] [location=1398620..1398865] [gbkey=CDS]
MSEELRGTLTPEKLEGLDEWFRLEAPEACRNASRLPFPYNQRILRHFRRMREEERPLPAIAGFLRHGLDEIYDILRDYQS
A
>lcl|NZ_CP026520.1_prot_WP_241688802.1_1196 [locus_tag=PC41400_RS06285] [protein=AraC family transcriptional regulator] [protein_id=WP_241688802.1] [location=1399000..1399671] [gbkey=CDS]
MDVLLIISHQLKEDEFERVQRLFSHMSIIAESDFKAVERSLHRQEKAHKLFVFLSFDESLPEEWKIEELQLQSPIMVLLK
VNKRDLFVFYSFLDSVGLLSKPSADVSAKPEISPMLYQSLEYIEQNLCENDLSLEKVASSIFVSRCHYSRMFKEYFGFGF
KEYVMNKRIQKAKAMLQKGVPVTDVCYAVGYGDLTHFGRVFKRLVGINPSVYRGQRREAPSCG
>lcl|NZ_CP026520.1_prot_WP_042229658.1_1197 [locus_tag=PC41400_RS06290] [protein=hypothetical protein] [protein_id=WP_042229658.1] [location=1399692..1400633] [gbkey=CDS]
METEPIRSRYFNRLTIGPDGRSVRKTSRHDKKLRDEISWYLNLPEELQTYIPGILGYSLQDQVYLCLEYIQAPTLSELYM
SGRMDQEGWAGVFARLDRLFGRFASFPAELAPACLHEMYIGKTQTRMAAFLNSNERARRFYNQGYYRLNGKMVLCPFRIF
ESHLGGFRSLLANPAIAVLHGDLCFSNVLYAPDTQAVKLVDPRGSFGNPGIYGDRRYDLAKIRHSLNGYEHIVRGRYSLQ
TEGNDIRLEIPFTDGQIRLLRDWDRHLGGGLRDVAKIEALLFLSMLPLHRESPERQLALYAVATRLLYEIFAD
>lcl|NZ_CP026520.1_prot_WP_042229656.1_1198 [locus_tag=PC41400_RS06295] [protein=hypothetical protein] [protein_id=WP_042229656.1] [location=1400715..1401002] [gbkey=CDS]
MNQTAAKFVMATAILVSPAAGTALKASAQTAAVQAQPAVQPAGQAAQAAQAVQSATSGTLSQTGVSGVKTESREKLQEFG
GANYFTPSVVGKTPA
>lcl|NZ_CP026520.1_prot_WP_042229655.1_1199 [locus_tag=PC41400_RS06300] [protein=hypothetical protein] [protein_id=WP_042229655.1] [location=1401187..1401456] [gbkey=CDS]
MKEITSKFIIATAILAAPAAATALQVSAQTVAVQPAASHEAASAVQTAKPQVQAAETSKASVYESARIGSIEEYTLAGPS
FGPPQKTMA
>lcl|NZ_CP026520.1_prot_WP_042229654.1_1200 [locus_tag=PC41400_RS06305] [protein=hypothetical protein] [protein_id=WP_042229654.1] [location=1401515..1401793] [gbkey=CDS]
MNQTAAKFVIATAILAAPAETALNASAHAAAVQSQPSKSSESVSTAVTAYRPSIAAASETKPEPQSRASAGIVELVEAEP
SFTFNAVIKRLA
>lcl|NZ_CP026520.1_prot_WP_042229653.1_1201 [locus_tag=PC41400_RS06310] [protein=hypothetical protein] [protein_id=WP_042229653.1] [location=1401874..1402446] [gbkey=CDS]
MNHRMFTCLDLRPCLNNCGTTGSHNYHEGYLSLARASFPEESFPFDTPFEFKQIPFVLFKRHTFDNIVTEGQSIEFPRQR
VTKIHLLGTSSDANMADYVYLMLDEREVSKQKILLSHFISEVPFPRNECVIETSCLHTMDGPLTHRSARLWYDGIELDRP
VEMNRITFEDNPCMHVFGLTLESESIPDSR
>lcl|NZ_CP026520.1_prot_WP_042229652.1_1202 [locus_tag=PC41400_RS06315] [protein=hypothetical protein] [protein_id=WP_042229652.1] [location=1402550..1402822] [gbkey=CDS]
MKETTSKFLIATAVLVSPAMGTALDVSAGQPAVIQTYSALADVQQDRAVSASEDTYASLQTAKKASDLRLQEPGEYTVSG
VSSLCVKCVA
>lcl|NZ_CP026520.1_prot_WP_042229651.1_1203 [locus_tag=PC41400_RS06320] [protein=hypothetical protein] [protein_id=WP_042229651.1] [location=1402918..1403202] [gbkey=CDS]
MNQTAAKFVIATAILVSPAAVETLNASTTVQALTAQSTAADSLRAEPASGAFLSGSTLAGVQTTPSEAGKDVRQEAAYTV
PPPDSTCAGFALKC
>lcl|NZ_CP026520.1_prot_WP_241688803.1_1204 [locus_tag=PC41400_RS06325] [protein=BtrH N-terminal domain-containing protein] [protein_id=WP_241688803.1] [location=1403327..1404340] [gbkey=CDS]
MSFTLYKETRLNCFYVNIAALLKRLPVEEGHLVYLFNQAGLFYRDTEDEDEFQISPYYRQLDDFVYGLFGARLHKEAASA
GNEYMRMLKAALANGKSAIIESDIYYLPYSGFYKRTHFPHALEVIGCREGTFEICDHYYEYHGFLEESELQAVFDASLNS
AFYKRLYVWFLVRTDGPPPEELTPAFCKSVIRDNQANMTGAITYDCPPGSYAGLAAFVPICDKIRSLKTIEEDAVKFKSA
SYLYEGFKEIANSRYHHHLFLKAIGEHPLAESYLEADQSWMVAANLLMRAATGNQFEAFEERVLARMERVKQKEADNTAL
LQQWLDRRREGPSDDVR
>lcl|NZ_CP026520.1_prot_WP_042229650.1_1205 [locus_tag=PC41400_RS06330] [protein=HAD hydrolase family protein] [protein_id=WP_042229650.1] [location=1404447..1404794] [gbkey=CDS]
MRIVLDLDGTICTLKKEGESYADVTPLPGSVEALRRLKEEGHRIIIHTARHMKTCGGNVGEVLARVGSLTLDWLERYEIP
YDEIVFGKPYGDVYVDDLALVYHSWDEVLERLNIG
>lcl|NZ_CP026520.1_prot_WP_042229648.1_1206 [locus_tag=PC41400_RS06335] [protein=glycosyltransferase family 2 protein] [protein_id=WP_042229648.1] [location=1404791..1405534] [gbkey=CDS]
MKPIQIVIPMAGRGQRFRDAGISVPKMLIPVAGKPMLQWALESLTPHFPGNTILFVCLEEHLEQGELETVIRSCCGNAVI
RSLPEVTGGQAETVLSTEACLDPELPLLIYNCDTYTESSIGRTIRTRGNEADGIISVFTSASPAYSYVRTDEGDRVAEVR
EKRVISNRATTGLYYFRQAGLFLESAEEAVRFGTAEGETYVAPLYNRLIARGFTVVADQADRCFPLGTPEEAASFTRNYR
KRESDGA
>lcl|NZ_CP026520.1_prot_WP_042229646.1_1207 [locus_tag=PC41400_RS06340] [protein=glycosyltransferase] [protein_id=WP_042229646.1] [location=1405531..1406739] [gbkey=CDS]
MTKRIFFGLSGGLGPVLRCLPVAEHFRRSGHAVAFSIYGEGSAAFIESRGFKHLPDDDPTVPSAERMIAGGPVFYNLDHY
YAGAGLLDPDFVQAWIHHRIRMLEKFNADLVITDMSPHTAIAARYLGIPVASITQSCFHPDGQPMHYWGRPPRNLPRVTP
VVNRVLNRLGLPDIRRMEQLNAGTLNFIPGIPEMDPVYTNGVRYVGPIEYPAAGPAGARLPDEPFVLVYPGRLQDTAGAT
GLQLLNHVTEAFGGKAEAVVIAASDGLPERLKAGLPANVTVIPYFNSHMLQNCRLFIHHGGHGSCLSAIMRGIPSLILPT
HTEREFNARKVQETGAGEYMLPGTFTADHLYKLALYIIEDGYKERAGELASLIRSRKYGGAREIYKRCMSLLNRRAMALE
EL
>lcl|NZ_CP026520.1_prot_WP_042229644.1_1208 [gene=acpS] [locus_tag=PC41400_RS06345] [protein=holo-ACP synthase] [protein_id=WP_042229644.1] [location=1406736..1407113] [gbkey=CDS]
MIYGIGVDLQNIAQLEEAVKRQGESFLRKVFTDRERAYCSRHTSPAQHLAARWAVKEAFLKATGSGLGGQYLFKDLEVVN
KPSGQPELILHGKAQEDCKRLGLHIYISISHSGEYALGQVVLCKA
>lcl|NZ_CP026520.1_prot_WP_042229643.1_1209 [locus_tag=PC41400_RS06350] [protein=VOC family protein] [protein_id=WP_042229643.1] [location=1407283..1407660] [gbkey=CDS]
MTQTLTGIDHVQLPVSDLKQSIEWYGKMFGCTLLDNYGECAMLKLENGPNLMLWQSSDYSSAHFDKDGETMPVIFFKSNE
IDLLASRLEEEQVRIVSFDDGGFAKFLKFYDINNNFIGVLQLAEA
>lcl|NZ_CP026520.1_prot_WP_042229641.1_1210 [locus_tag=PC41400_RS06355] [protein=sugar phosphate isomerase/epimerase] [protein_id=WP_042229641.1] [location=1407859..1408722] [gbkey=CDS]
MTQMFHRSKVSCQLTPWGEAYRTGLKELAALDVAYVEIGSRLLLPFEREMDRWRELLDETGHRVSGVFEFGHFAGWNRRR
EIYLHHDRLARMLESAGIGRVILGPGIRLKRDWTGEDRLRMAAMAAEINKRYGFHGVAAAIHPHISHCIYTREEIDYVME
RTPGDLGLVPDVGHLHEAGIALRDLLERYGQRITALHVKDVRPVEGDPGSARPGSRKTVFTELGGGMLDLSSLLRDLAGH
DFSGFLTLEMDEVSCPPGEAVRLSLDVLKSESSKFSQERKKIDGHVD
>lcl|NZ_CP026520.1_prot_WP_042229640.1_1211 [locus_tag=PC41400_RS06360] [protein=ABC transporter ATP-binding protein] [protein_id=WP_042229640.1] [location=1408712..1410460] [gbkey=CDS]
MWTKLPILPYFRLVGPYLKQHKSRFALLTLLLAANIVVQLTMPWLLMKYFDSLNSPEPAERFIFLFILLGVCYQILSLAS
DYVGEHVGWSAANKLRSRLMRHCLGQDIAFFHHYSPGELLDRVDGAVNGLAEFFYKFVVRFTNNLVLLGVTLIVLYNIHW
SIGAVMTLFVAGSALVYYLLRDYALPHWVRVGETSSAFFGFIGEQLTNTEDTRSSGASGYVMRKFKALLNQWLPVHMEAG
RAFSVMWMSSMLLLGLSTILTFGVSSWLWHAGLAGPGAIYMIMMYTVMLVGPIDTIRLELGALQRASANIERIRQLFEIR
PSVGDPAEGGDTLPGGALSVEFSGVEFGYTPGEPVLHGIGFKLEAGESLGIVGRTGSGKTSIARLLLRLYDPQSGTVSLG
GTDISRVKLSGLRERVGIVTQDVQIFKGTVRDNLTFFDPSIPDGKIVSLLTELSLDSWFEALPQGLDTQLEAGGGGLSAG
EAQLLAFARVFLKDPGLIILDEASSRLDPATEEKLDRAVRKLLRGRTAILIAHRLSTLRHVDRILVMDGGRIAENDLQGA
LLEDPDSRYSRLLRQSLRNRTV
>lcl|NZ_CP026520.1_prot_WP_042229637.1_1212 [locus_tag=PC41400_RS06365] [protein=ABC transporter ATP-binding protein] [protein_id=WP_042229637.1] [location=1410457..1412202] [gbkey=CDS]
MSPLRLMWRLSVCHPRLFWKGAAIWLPFNLSPLIAGLLVKLVFDKLAGTGEEAGGVWFVIALLAVNALASAGSIVWGTRT
ETELNFRVKSLIRQNLLEHILKRPGARALPYTTGEAVSTFRDDVEELADLLRLYNLLPVKVLFAAAAGGILWSASPGITL
LVFLPLVFMLSLGQLAAASLQKRREASRKAESGISGFIGEVCGAAQAIRLQNAETFVSRRFRSLGEVRMRAVLKERLLDR
TVSAFLTNTTQTGTGFILLAAAFLSGKGELTVGNFALFVAYIGVLTGFVQYVGYLLKTYRQASVSVNRLLHLLQEASPDT
LLARRKLELKEEAPTPSPGLPASPSRLHLLETRGLTYLHRGTVRGIRNIDLHIPGGSFVVVTGRNGSGKTTLLRTLLGLL
PADAGEIRWNLEKVEEPDMFFTPPRAAYTSQVPGLFDGSLKENILLGLAEERLGEAVYQAVLDKDISQLEKGLETIVGVK
GVKLSGGQIQRTAAARMFARESSLLVLDDLSSSLDARTESELWKRLDRLRDTTFLVVSHRRAAFERADHIVVLKDGAIAE
QGTLDELLRSSGEMRRLWDRP
>lcl|NZ_CP026520.1_prot_WP_042229635.1_1213 [locus_tag=PC41400_RS06370] [protein=ABC transporter ATP-binding protein] [protein_id=WP_042229635.1] [location=1412244..1414016] [gbkey=CDS]
MAFLSIPKVPVLRYYKLLRTYLAALRWPVLALCALVALNIGLQLINPQVLRVLIDTGLSSSAAGIAGPALLFAGLALVQL
LATISADYAGEWIGWTAANRLRADLLEHCLMQDGSFFQAHPAGELIERIDGDVRGLSGFFAMLTVSLCGNLGLIAGVLAV
LFYENSWMGAAMTLFVAAALLTFGRIREWTLPFWVRVGEAGADFYACLGDQLSHTEDTRSSGAEAYALYKFRRMLRQWFP
RHMQAGFGFSLMWAASVVILGLGTVLVLALGTYLWTGDSITVGTLYMMLFYMGLLAGPIDQIRDLIGNLQRAEAHLERIG
GLLAEVPIIRDPEPGEEAPLGEGPLSVDFRGVRFGYGGESGSVLSDLTFSLQAGQSLGIVGRTGSGKTSVIRLLLRLYDP
AEGTVLLGGTDIRRVPLAQLRERVGLVTQEVRILSGSVRDNVTFFASHIPDERIVEVLDQLGLGDWYRALPSGLDTEIRA
QGEGLSAGEAQLLALARIFLRNPSLVILDEASSRLDPSTEEKVQQATNRLLQGRTAIVIAHRLNTLRGVDRILVMDGGRI
AEQGSRQALTEQPGSRYSRLLRQTSGKETI
>lcl|NZ_CP026520.1_prot_WP_042229633.1_1214 [locus_tag=PC41400_RS06375] [protein=ABC transporter ATP-binding protein] [protein_id=WP_042229633.1] [location=1414013..1415785] [gbkey=CDS]
MNTKQFLWRMAAYHPGLFWKGIFQWLPFNLAPVAAGLLMKVFFDQLTGHSASSDVTWFVVALLAANAAATLGSIRFASRT
ETALNFRIKGLIRQNLLNHILGQPGARAIPGSPGEAVSQFRDDVDQLADTIRNVNLLVVKTVFFLTALAVLFAINVPVTL
GVILPLLAVFALVQTRIIRMSVRLRPKGNGRRELSGFMSEVFQSVLAVKVAGAEDSVVEEFSRLSDDRRKQVSRDRVMSQ
LMYALLPNLVQIGSGILLLAGARLLGPGKFTVGDYALFVHLLLHMSKFIEYSGTSITQIKQVKGYLRRLNWLLQGSSLDR
MIDFKPLDLAESPPAAPPPAVSESRQFKQLQTKSLTYFYPGTAKGIKRIDLQISRGSFVVIVGRVGSGKTTLLRTILGLL
PAAAGEVRWNNRRIANPGEFFVPPRTAYTPQVPKLYNGTLKENILLGLPEEGLEQAVYDAVLEEDVRLLDKGLDTEVGVR
GVKLSNGQIQRSAAARMFVRGSEVLVVDDMSSALDAETESRLWERLNAREEATCLVVSHRREALRRADHIIVMKDGEIEA
QGTFRELALSSEEFRALLDEAGGTLPGNRP
>lcl|NZ_CP026520.1_prot_WP_042229632.1_1215 [locus_tag=PC41400_RS06380] [protein=PfkB family carbohydrate kinase] [protein_id=WP_042229632.1] [location=1415894..1416895] [gbkey=CDS]
MEPSRKEKLLNRLGRKPPGTTIVLGDILLDEYTACTRDYSANEHCVVLRKGPSLFFPGNALNVSLNIAALDGSAAILGAM
GTDTEADRLRSLAGDRIDSGGLIGETGRQTTLKTRLSDGDGLLLRVDREDNTPLSASAAGLLLKHADSLPAVSCAVLSDL
GKGVLPDSLIAGYIRRAADKGIPVLVDPAGENIRRYAHASVLFPNLAEFNKLTSRSCENPAEAARYAQRKLAEWEIGALV
LKAAEAGSYLITTEEIRHVPAYCKEPVCEVGAGDSFLAAFAVGIAHGLPLEESFMLSNASAAVSVSKAYTSTVSTRELTA
FIGQLKQWDDRRP
>lcl|NZ_CP026520.1_prot_WP_042229629.1_1216 [locus_tag=PC41400_RS06385] [protein=glycosyltransferase family 9 protein] [protein_id=WP_042229629.1] [location=1416908..1418404] [gbkey=CDS]
MADEAFRQQVKNRRDYLRSSRGKILVWTAAECEEDVMRLLSEKTPDLAAVRLRMPDLALFRAGGFTVLPVIDRNGLEALA
RFLQGERGRAEIESDSGANLTAAADSGRIVTAFVFHEPRTESVTGTGTGTRIETGSEAADYAHLIGDIPKQYEPAELYVY
RERFCFPFYFRAAEPAEKLDARTAVRGKPKPKAVIADWFLGYGDGVILLPLLRKFIDREARLGQEVHLIAKEKLASFFGT
FLQGCRIYRSPESPAVYEAVLRSSLYRKAILLPVSVSSPPMRHISGLFGAALGFKRPYPLPEPAPAVYPPLPAEQAEELA
VLRAAGCKAVGIQFHTGDAKRSWPEARAREFIALCREWGLALVNLTPLPYPAEGCSDWSMLPVEELFAAIEGLDAVVGID
SVCGHVAGFLGVPHITLWGRSFPHLQFPFTSDESHVSFRTLRANYSLVPQSADIADIPAELAFARLTDILEGRLSPGKEL
HTAEDTLKGQGIEWVSGR
>lcl|NZ_CP026520.1_prot_WP_042229627.1_1217 [locus_tag=PC41400_RS06390] [protein=hypothetical protein] [protein_id=WP_042229627.1] [location=1418401..1418841] [gbkey=CDS]
MSTIVYIWDRTRDGADLENLRAELSRTIREYGYSVSAASDNEGLADFLLLSEAETTEGNAENKSVLIVTDDEEVLSDSVC
YRSAEEIWQTPFDVGGIVLLAEILGFAHRFTGFSYSPMCLVHVPSRSLSYHRLVSYWIHKSGRKVR
>lcl|NZ_CP026520.1_prot_WP_042229625.1_1218 [locus_tag=PC41400_RS06395] [protein=adenylyltransferase/cytidyltransferase family protein] [protein_id=WP_042229625.1] [location=1418857..1419342] [gbkey=CDS]
MADLPELKAELARVRAGKSGERTKTVVMAGGCFDIFHIGHLDYLQAARGMGDILVVAVNSDASVRRIKSRPPVFEAGQRA
GIVAALEAVDHVFIFGEDDCCESLRVIRPDFFVKGWDYKDKTFPEMAAAREIGCEIKIAGEDKKSSSSRLRSILRGERRF
L
>lcl|NZ_CP026520.1_prot_WP_042229624.1_1219 [locus_tag=PC41400_RS06400] [protein=carbamoyltransferase C-terminal domain-containing protein] [protein_id=WP_042229624.1] [location=1419339..1421060] [gbkey=CDS]
MNILGLGGSIHDYSACLVRDGEVRCMMEEERLTRVKHCLGLGGKVFRCRAADYCLQAEGLDAGALDLVVGNDLIDPVYYM
KYADSIQLINHHLAHAASGFYPSSFPSAAVLVADGRGSFTGFPAERETVTYYFGNGSELSEIKKVSGTDTLDFGAAHDSE
SGLTDADMSNSVGGFYERVTTEIGFGLLQEGKTMGLSPYGTARHTDAFGSFYSMDDEGNFRQTLDQVKELRMYIRRTLEA
AEDPEKFQAKADLAYAVQYHMERILVKACRYLYNVTGSRNLCLAGGVFLNSVANYEILRQTPFEHIFVQPAAGDAGTAIG
SALYGYHVLGGRARTAVNIPFSPYQGKSYGPDECAKAAGAFADRIERVEAADVSMKTAELLAEGHIVGWFQGRSEIGPRA
LGNRSILADPRQAAMKDTINTRIKHREHFRPFAPAVLEERQADFFDLGCPSYYMLLVPPFLPGKGEETAAVCHEDRTGRV
QTVSRGLNPRFHELLVHFESLTGTPVLLNTSFNDNGEPIVETPHDAVSCFLRIDLDYLVLNGTIYKKRKEGGAGREKSGE
PPSGHAGSEAWPS
>lcl|NZ_CP026520.1_prot_WP_042229623.1_1220 [locus_tag=PC41400_RS06405] [protein=SIS domain-containing protein] [protein_id=WP_042229623.1] [location=1421002..1421688] [gbkey=CDS]
MKKAVSRHLDTLVRKHGRLEDSRTAMGEAFAVMEGSFRNGGKLLVCGNGGSASDAEHIVGELMKGFRLKRPPEAASLKRL
KGRMPDTAGLPGEPVPPCLYGLQLGLPAISLVSQTALVTAVLNDLAADAVFAQQVLGYGNPGDVLLGISTSGQAANVTRA
MEVARTLGLATVALTGAGKGRIRAWSDVVLASPSSVTDEIQEDHVKMYHTLCLMLEEEFFGPESETEG
>lcl|NZ_CP026520.1_prot_WP_042229622.1_1221 [locus_tag=PC41400_RS06410] [protein=carbamoyltransferase C-terminal domain-containing protein] [protein_id=WP_042229622.1] [location=1421690..1423627] [gbkey=CDS]
MQDGYYLSAYLHIDPLDHLLGTIIRHDQNAALFQKSGRTVKLIRLWELERVTGIKEHYVSFHSPEQAKRVLNELLKPLGL
SLRDMVEIWGTPGLQTAADYHSVQEYRDLSYHSISHLFSAVMADSELFYNKRILGFSVDGGPDGVVDTRERTEKYFFTGG
YINEGRVNLFPAESPGMLWGWAKFYFRMREGTLMALASASTSEIYGERTAGFPVLDGLHDMWESDCIRDLVNRVEQLTDE
DAGRLFNGFDPRFSRRENKISMIMKIVQQVSMEVMDANIDRSLAAYSIDPSETYLALAGGFALNCPTNRHLMNKYGFAGF
IAPPCVGDSGLSLGMGLYAFHKKLRGEPFEFKLAHAYFGEPDSSLPDILSSEAYAPFIEGAGPMTEQQAARDIQRGLVLW
FDGESEIGPRALGHRSLLGDPRTEESKRMVNEVKQREWWRPVAPIILEEETAEWFETDYPSRFMLHTFKARPDKAPLIPG
VLHLDGSARVQSIGRLENPRLYGVIGEFRKQTGVPVICNTSLNDRGEPIVNMIHEALNFALRKGIPVAYINGMRIRLRNH
GDYAQAHEGPYPRRIGMNALSDREKEAAWKELNPWDVPRDVLYHYRDPNEQEPAGVYDIRSREDAERLCREVRLVQRHKK
LFRTI
>lcl|NZ_CP026520.1_prot_WP_042229620.1_1222 [locus_tag=PC41400_RS06415] [protein=carbamoyltransferase C-terminal domain-containing protein] [protein_id=WP_042229620.1] [location=1423652..1425442] [gbkey=CDS]
MLILGLGGSIHDFSACLLHEGKIVCAIEEERLSRSKQAFVDASTFRCKAARYCLDYAGVTEKDVDLIIGADNIESRYYAR
YVPGIRLMNHHLAHAASAFYPSPFTEAAVLVADGRGSYLDRENKIVETVTYYAAGGAEIREIRKVAGRESDDFRNVSNSA
GLFYLAVTEEIGFGPMYDEKTLAVSAYGTDRLVEPFRAFYSLDDEGGFSQTLGQMNELRSLIRAELGRSESGGELFRTKA
DAAYAVQVHLERILIHACRYLHKRTGSDNLCLAGSVFLNHAAVNRLLRETPFKRLYLPPAPGDAGAAVGSALYGHYVLGG
HHREALLPATPYLGKPYPAEAMEEAARKAGDFLEEVKTEDINGQAARLLSEGNLVAVFRGGAEFGPKALGNRSILADAGF
PGNISRLNSAKKREAFRPFGYIVLEEDAGKAIPPQGERRFQAHAAEAYASGRLDDAESVGASPADRTDPAGPVNPASLAD
LADPATPVTPANRTDPVGPVNPRDPATPVNPRGLTTEGAAQMHAISPGLNAEMAALLESYKRITGCPLLLNTSFNLDGEP
MVETPAEAVRCFLKAELDVLLMGNKLYRRKKTEGEA
>lcl|NZ_CP026520.1_prot_WP_042229619.1_1223 [locus_tag=PC41400_RS06420] [protein=hypothetical protein] [protein_id=WP_042229619.1] [location=1425445..1426548] [gbkey=CDS]
MKHLDDKLLQDFRRFYESIPEKNGVYYLFFTTGLLHWAAKSLQFVPDTVPVVLIGAGFTEEEQIWVRQEIRRPFFCFGEK
VDEGIVWHYLTAVNREDFGWLDIDCFVLNDRLFGEMARVPRSASMNCVWSYEYGKRTVPPAHFDFRVLRTFFLFINQDVI
RELDDGDLPWSPRPFLLDAAPVSSSFKRYDTLHVLDDAHKRTLARILPENGDGTPIFPAQIIKKDGVVTRHLDTLVLYQL
LAYSRGYPLHRVRCLDEDETALSDEVVHIGSVSYYRSFADPGYEVVSGEERARYNEIYERVLLLDYTIVSQFERRLPRSY
SVYRRMLERDIKRLGISSEHVKEAALQKLTGQGMSREAAERICGDPL
>lcl|NZ_CP026520.1_prot_WP_042229618.1_1224 [locus_tag=PC41400_RS06425] [protein=NAD(P)-dependent oxidoreductase] [protein_id=WP_042229618.1] [location=1426545..1427480] [gbkey=CDS]
MRKAARKVLVTGAGGYIGCSLVPELAKRGYEVLALDKYTRSQDRHFIHSGVSRIYADTRTWSGEGLEGIDAVIDLAAISH
DRRGEITPHETLDINQFARYRTAKKSREAGAGRYILMSTCGVYGFRDDPGGADEKAVPNPLTTYARAASRSERDILLLGR
EGFDVTVARPAILYGYSPRMRFDLAVNAMVRAAWQTGTITLQGGGGQWRPFVHLKDAVEAVCCLLEAPAEKVSAQIFNIG
SGGQNIRIGDLAELVAAVLPIPVEIRREGPADPRSYRINADKIGAMLGWKASHSLKEGIEELYGKLARGLV
>lcl|NZ_CP026520.1_prot_WP_042229617.1_1225 [locus_tag=PC41400_RS06430] [protein=phytanoyl-CoA dioxygenase family protein] [protein_id=WP_042229617.1] [location=complement(1427688..1428494)] [gbkey=CDS]
MKSESANKALPDLSSGYTLTQEQIRYFRDNGHICLKQVLTPAEITAYRPRILEAIRDYMTNTIPSSGPDDSGTVNLRLRH
PDVGRFTASRRIAKIAADLMGVEGVRIYRDSAFFKQPGAAATPMHQDNNYMLLDTDRSITAWIPLREVTASMGSLHFISR
SHTLDNRGKTAKQNLKAAYRLGLEEMTYGGIGLGDVTFHAGWTLHNAPPNETSDTREAMTMMYYEDGASIVNPFETDPTG
HLNQYFGGQKTGETADTPFNPLVYKKGS
>lcl|NZ_CP026520.1_prot_WP_241688804.1_1226 [locus_tag=PC41400_RS06435] [protein=nodulation protein NodU] [protein_id=WP_241688804.1] [location=1428672..1430648] [gbkey=CDS]
MWRGESMQDGYYLSTYFHIGPLDYLTGTCIRHDPNMALFRKEGGDIRLIRYWELERFTGHKQHRQSFYSREHATGVVNEL
LGPLGLSLEDMTEVWGTPGLATCSDYHSLDDFPDLSYHSISHLFSAVMADSRLFYEGNVLGLAVDGIPDDVVDREAGSKP
FFAGCLVRRGKLELFPVQSPGPLWDWSSTYYNLREGTLMALAYASECAVNPVPLELPPAYDRNFNIWEHDYLRPLIEAAD
KLCERDARENPACTGFDPRFTERENKISAVMKQVQEASIALMERNVEAILDTYGVKPEETYLALAGGYALNCPTNSHLMR
KYGFKGMIAPPCVNDSGLSLGMGLYAFYRKMSPARIRFRLGHAYHGAAGGPLEEVLRSGTFAGCIGSVTPLEENRAAEDL
CHAPVVWFDGPAEMGPRALGARSLLGDPRSPAAKDALNAVKQREWWRPVAPVILAGEVGEWFEEGCPSPYMLHTFRVKAE
KRELIPAVLHLDGTARVQTVEDAPSTRNLYRLLRAFKEATGVPVLCNTSLNDKGEPIIQDIGEALHFALVKGIEVAYING
VRVQLVNHGSYRARAGAASFPRRIEFSRVSPEEREAAMLRHNPHRIPLEILKFYMERIERGGIERYDLTDPKQARRLIRE
VQLHRSRPGEREARDPEA
>lcl|NZ_CP026520.1_prot_WP_042229615.1_1227 [locus_tag=PC41400_RS06440] [protein=phytanoyl-CoA dioxygenase family protein] [protein_id=WP_042229615.1] [location=1430679..1431470] [gbkey=CDS]
MTAYSLSVQEKQFFAEEGYLVLKGVYSPEEVNALKEEYHKVWLDSILDKKIVQDPEKPLTSLFPRMKDLHKDHALVREFV
FKEKAVAALEELAGEEALLVSTNFYFKPPGSTGMPFHQDNYGIGVLPGTCYAIWAGLDPADVSNGGMRIVRKTHTQDLQV
PQKVYTTAEDTFGGYVQTLAVPEEHELIELRTDPGDVVIYHGNLIHDSADNASEVLFRHSIISHFAGTSAEKTTLNFNYL
MDKEGQRVRKRMNAGANLGRKGE
>lcl|NZ_CP026520.1_prot_WP_042229613.1_1228 [locus_tag=PC41400_RS06445] [protein=phytanoyl-CoA dioxygenase family protein] [protein_id=WP_042229613.1] [location=1431474..1432307] [gbkey=CDS]
MNEKLPDDKLAAFAGEGYLVLNDVYTREEVSELIRQFDLHWVRLTASGQIIQNGNRPLESLYPRLRDYHRKNEKIKRLSL
KPQLFDYLEQLIGEEALIISTSYYFKSPSTRGLPLHQDNYAFGVSPGTTYAAWISLDSSDEDNGGMTFVPGTHSLDLQVP
EGDASDVRRYFSDEGQWLKDYETAEMVKVATNPGDVIFFNGNIVHGSSDNLSRHRFRRSLLIHFTGVSVEKLALNFNNLM
DKTGARVRRRLNTDTKITEGQDSVFSIQEADYFAGWR
>lcl|NZ_CP026520.1_prot_WP_241688805.1_1229 [locus_tag=PC41400_RS06450] [protein=tRNA ligase] [protein_id=WP_241688805.1] [location=1432316..1433614] [gbkey=CDS]
MEKAGDRMFIRVPLDKVAKAGDLIQLMEKIYYLSDRITNVSRDEAALTIEYGGSDEEEVVRRLDALCASMAGRRKLPYRK
VKDNRRDGKAGGRAQAQTGALSSRERDSERSEGSGVSERSEHSESAKEAHVLLEQALVKLLRGRAHAYGAALRRYPAMMP
ESVLQKSGYIRNFPQNVYAVGEFRHQFDTMQEIRTRLEDNAPLHELLQPSGMYLQPCVCYHVYEERSATRSGSPAGNGSG
PGSAGAVSSGPKLALYSAYGPCFRHEHASRIDDSRLREFGMFEIVYIGGAQEVRDMRQKLLEDTWRLFGQLGLEGYVETA
SDPFFLPEDTDRRLFQLAAESKFELRFAPGGESASDYAVTSFNVCGDVLCKAFGITGEPAEYGDSGEAGEAGKGVPLHSG
CTAFGVDRWVQALAETHGYEAGSWPQLIREHL
>lcl|NZ_CP026520.1_prot_WP_042229612.1_1230 [locus_tag=PC41400_RS06455] [protein=hypothetical protein] [protein_id=WP_042229612.1] [location=1433616..1434641] [gbkey=CDS]
MGTRHLLPVVESPGFETECYHNIRLSLLLAHARTLPWYHSHFMNLTLKSTGADLFPVVRFEEHLDVYSEVLEETPLVRCE
DWVETARKNLGEGRYMIVYFNWRNIPCSRYYNKQDMFHEALVYGFDDERGVFHFLAFDVNGKGYGSSEVPYGLFGREMTR
LSDEDLHAQRWFAFYGFPASVIRLKDGLASRPDTRNMYFALERGRVRGNGSPDETFAMGHYVNEVLSVYFRQLAEGRPLQ
VKEFIYWNIIVHKMILHKSLMLQRIIYLQTGTDSSQLDLLKNLYIKAKTEMDRVKWTSIKYQRTKASAYLHQLADNFQTL
YELEKRASAMLMEFLTLSHLK
>lcl|NZ_CP026520.1_prot_WP_042229610.1_1231 [locus_tag=PC41400_RS06460] [protein=acyl carrier protein] [protein_id=WP_042229610.1] [location=1434667..1434918] [gbkey=CDS]
MDQTAMEAVIIEHVKQILPQQIEIELTTKLLETGIDSMSFIRLVVLLEEAFNIVIEDEEILLENFTSVETIRNLIRKCLD
SKG
>lcl|NZ_CP026520.1_prot_WP_042229609.1_1232 [locus_tag=PC41400_RS06465] [protein=hypothetical protein] [protein_id=WP_042229609.1] [location=1435047..1435580] [gbkey=CDS]
MKTVSSKKNAKQVTMTLENSNSKTYLELLDGRSEELQFTQVAPTEFTVNDSEFSLKSGINVELGIKNVDLVATSSVIWPG
QTIRVRGGLHGQGAAMKAQATIPFNRKMADGVQGESWLYWVIETPEGEFHNKQAIHMKGTLKGLPPKNATFYSDSVTPLF
DSEDNQVGTVYGCLQSN
>lcl|NZ_CP026520.1_prot_WP_042229607.1_1233 [locus_tag=PC41400_RS06470] [protein=NAD(P)/FAD-dependent oxidoreductase] [protein_id=WP_042229607.1] [location=1435659..1436837] [gbkey=CDS]
MYDVIIVGARCAGASLASFLGRRGHSVLLVDQFPYPNATTSTHILGESDVYERLGILGKMESAGAPFLTRMRVDLEGTVF
ESDITVTKRALGLRRELLDPLLLQNAARHRNVEVRLNTRVTKILLENGRAAGIECRRKDGQLQRFAGKAVIGADGRYSAV
AKGVRAPLETETERPLHGVAYAYLSGVQRLPVPAVEWYWKGSSVVLCNPIDGLMHCAAIMYSPEEQERMARLGRGEFMER
LLEIRTLAPRLAESGLEGAVRGIPPGKSFIRKAYGCGWALAGDAGAFLHPVAGVGIDNAVCTAEVLADELHAFFTGAKEW
EEAMRSYEEQRNNRIYPQYRHSLATLETAGDPFTETSAGALKMLATFPSLVKKIGLQAEAVISFLSGGQNHD
>lcl|NZ_CP026520.1_prot_WP_042229606.1_1234 [locus_tag=PC41400_RS06475] [protein=PDZ domain-containing protein] [protein_id=WP_042229606.1] [location=1436830..1437768] [gbkey=CDS]
MIECPTDRTRNVTYLEAAPEQVWWSIATTEGTNTYLTYSSSTTGDPSLPKAGDVYTLNYGDIENQSVVVESVPNRRFVLS
DAYQSLAPDGKAEIFYVNTAFTLEESDGFVKLSLQVQGFAEDTYGQWLRECLEMGWRRSLMNLKSVLELGLDLRTELFSY
PRLGVTNCTVNPEQSRETGVSVGGGNYLLEVFPNSPAQKAGLRAHDVILGIGGRRTAVYGDFVRVISSYSGKKGPVSVTF
SRSGKRHETTVLLSVEEQFTGLNKESLSLEEIKQRRELIAKSRSASGAIWTGTENGMKTAAENGEDGAAGAQ
>lcl|NZ_CP026520.1_prot_WP_042229604.1_1235 [locus_tag=PC41400_RS06480] [protein=phytanoyl-CoA dioxygenase family protein] [protein_id=WP_042229604.1] [location=1437790..1438641] [gbkey=CDS]
MIGEDRSATAGLDREYRLTGEQISRYRRNGHAYLEKVCSAEEVKAYRQAVTEIYPAFTADSYPPDEREFYGVERAFPTVM
NMWERSSKVRTFTFARRFAQIAADLMGVDRLRLYHDSAIFLEPGGGAIPWHQDSPYMLQMDPDMTMTMWMPLVDLPEELG
SMRFMSGSHELPADKHTNLLLQAYRKGQAEVHYGAMPAGDATFHSGLTLHYAAANRTSIVREVATIIYVADNLQIVEPDT
EARAYHLRRIFPGMKPGDTAVSPWTPLVFDRFAESGSRTERKG
>lcl|NZ_CP026520.1_prot_WP_042229602.1_1236 [locus_tag=PC41400_RS06485] [protein=phytanoyl-CoA dioxygenase family protein] [protein_id=WP_042229602.1] [location=1438643..1439467] [gbkey=CDS]
MENTHGFDALEQEWSVSREDIDGFRRDGHLLLKQLASEEDIGGIREACRREIARLDDGLRLQVMNVWEKDAAVKAFVFAR
KFSRIAADLLGVDGVRLYFDQVFFKKPGAGAMPLHQANQFMLELDPDSVLSMWMPLTDIPGEKGPLSYVAGSHLHEKLRG
AKNPLLAAMRLGLSEKPCGPMQAGDASFHTGWTLHGAYGNDTEITREMLIITWFAEDARITDPADSAGRNPQLQSLFRGL
EPGGKAQGRRLPLLFYRPYRDVVKLVHQAEEDAE
>lcl|NZ_CP026520.1_prot_WP_042229600.1_1237 [locus_tag=PC41400_RS06490] [protein=phytanoyl-CoA dioxygenase family protein] [protein_id=WP_042229600.1] [location=1439464..1440261] [gbkey=CDS]
MNGAVHDDLPPLSDNYALPEGLIAQFRANGHVCIRQAAFIHEISAYRPHIITAVEGYTGDRSVRPHKQLYMNLWEEDTRV
RQFVFARRFAKLAADLMGVTGVRLYLDQANFIEPGASVTPWHQTEAYMLELDPAQIITMWMPLVDLPEPGGPVTYLSGSH
LLPGGASKNPLLEAKRAGLPEENYGPLRTGDATFHAGRLLHSAPGNGGAFTREVMTVTYFADGARILDPEDKSGRAPHLR
RLFAGTGAGRPAEGPLTPLLYRRLF
>lcl|NZ_CP026520.1_prot_WP_042229597.1_1238 [locus_tag=PC41400_RS06495] [protein=phytanoyl-CoA dioxygenase family protein] [protein_id=WP_042229597.1] [location=1440276..1441109] [gbkey=CDS]
MENSLFTANLHEEIVLSPGQIEEYRSKGHVYAGRVASEEDVAALRPLVRELVRTCVKEDRPLEERDDFGKAFLQLINLWQ
RDEAMKQFVFARKFARMAADLLETDGVRLYLDQIFYKEPGGGPTPWHQDGAYMLRFKPDKLITLWLPLTDIPDDMGSLRF
ISGANEIERMKNSGNILLDAVRRGLPETGYKGMKAGHATFHSGWTPHSALANPTDSMREVLTITYIAEDAVIADPEDNEA
RRKHLETYFPGRKPGERADSPLNPVLYSRDGDCPVCV
>lcl|NZ_CP026520.1_prot_WP_241688806.1_1239 [locus_tag=PC41400_RS06500] [protein=acyl carrier protein] [protein_id=WP_241688806.1] [location=1441214..1441465] [gbkey=CDS]
MSAKESAKDQIRLFLSDQGCGMLPVEEISEEKRLFDVGFDSLRFMELVVLIEEHFGIQWPDEALEISGSTQVGDIVQALE
GCL
>lcl|NZ_CP026520.1_prot_WP_042229595.1_1240 [locus_tag=PC41400_RS06505] [protein=FabA-like domain protein] [protein_id=WP_042229595.1] [location=1441462..1441902] [gbkey=CDS]
MSRRPDYGHLLTMLPHQFPLRLLDEVKDYSPGKFLVAGCSPLPLKIYFGNTDTIPPTLLIEGLAQACVILTQLETAPLEE
DEVPLLGAIKAKLLMPVSWQEPITYKVEPVRILSKKAIFTGTLYRSDGTMAVSAELSVAVAATSNS
>lcl|NZ_CP026520.1_prot_WP_042229593.1_1241 [locus_tag=PC41400_RS06510] [protein=FabA-like domain protein] [protein_id=WP_042229593.1] [location=1441920..1442339] [gbkey=CDS]
MPAEAYLHEAKPWTMVDRIIGESDRAISTLKNISGSDFYLIGHFPSYSVYPGMLMVECILQSAELLLRRQSPGKPGDCRF
GEIASRFLSPVVPGDSVRFDVGFTAGGSELAAVGKVGGRTVIHCKMKLLREEREKDERR
>lcl|NZ_CP026520.1_prot_WP_042229590.1_1242 [locus_tag=PC41400_RS06515] [protein=alpha/beta hydrolase family protein] [protein_id=WP_042229590.1] [location=1442326..1443135] [gbkey=CDS]
MNGDERLVELYDQGKLAGILHLPPEHLRPCPVIIYCPGKNGERYEVHRLAVKYARRLAEQGIAFLRFDYYGIGLSDGHYF
DMTTSGKISNVLKAYEFVCGHPDLLEREAVFLGFSDGARIALMAAQGSGIEKLVLWSPLFFEFGGNYPSGRHPRFIRHPE
AGKLVMPWAGLWVGMDFYRDLQSMDTEAALESYSGSSVLIYGDNDPLILEEMELSESRRSAIYRHSAQNRVVRVPDAGHL
FTSRPLEETLMEHTTSWLRTEHLVRGGWG
>lcl|NZ_CP026520.1_prot_WP_042229588.1_1243 [locus_tag=PC41400_RS06520] [protein=alpha/beta fold hydrolase] [protein_id=WP_042229588.1] [location=1443139..1443999] [gbkey=CDS]
MWKQRRFGRDRGIYGLVEEPDGPGNDLVVMFAGLGQAMSEKNYFFSNLRKYLARSGVRVVQFDYRGHGDSFGELGDTTVS
GMKRDALEVLAEVLEERPAERIFLIGNALGGIIALQTAAELTEKGEASPVCIPVLLAPPFRAFPEKEALFGREALQALER
EGKLDAQLLVPGFDYYTLSDFNREPYDFFMAWGAHMLYLHGQCISAGMLDELLELDTARLVKEYGGPLHILCGSEDEDAR
TTAEETDRVILHRMRGYRYFYQHPAAMDQWITEAASIVRSDNGERE
>lcl|NZ_CP026520.1_prot_WP_042229586.1_1244 [locus_tag=PC41400_RS06525] [protein=hypothetical protein] [protein_id=WP_042229586.1] [location=1444004..1444972] [gbkey=CDS]
MEETRKTIAQTILSQKTDWRFPDGEGLSQRTLDCRQLCMMEVLKWKGERQLTPVFLNAFDCESNGLMLRRSEPEPGRGWR
LTKYSPLNTEGLYPLIDSVFDHEGYCLLFYNAKEAVFSSYHGKHDIVHWSLLVDYDEAGVTLVDDEGAPAFFNGYIGKVP
WQTLMSAHASSEKGGVAILGRRPGYAKTWEEEFLSLVRASVHNMVDQQGLKNLENFVKAVAESRAEALIPMLETLEFDVH
YFRRLRELWKAAVENGTVPDRYKAPGWVEELVYVCKLWSLVMGVLAKWKRQPDKDYSAKLTDYLGQALESEKGFYRELRH
LV
>lcl|NZ_CP026520.1_prot_WP_042229584.1_1245 [locus_tag=PC41400_RS06530] [protein=aminopeptidase] [protein_id=WP_042229584.1] [location=1444981..1446078] [gbkey=CDS]
MKEEFKRIADTVVSHSLSLRPGEKVMVKVRGDAADLTESLIARIYEAGAVPFLQTTDVEELKWLLAGASEEQIRQWAEHD
LGLVEAMDAYIGIHSEENTFEMEDADRTRYGYYVKHYLQPLTMTMASKPKWVLLRYPTPGMAQQAGIGTSKLRQIYASSC
LMDYGGLQDKAEPLRKLLGRTSRIRIVSPGTDLSFSVQGIPSFLCDGRYNLPDGELFTAPVAGSAEGRITFNVPTSCMGK
TFDSVALTFRQGKVTEWSADDIEALSGILKTDEGASRPGEFGIGLNPHIRRPMRSLLFDEKMAGSIHLALGQAYEMADNG
NTSGIHWDLVLCQLEAYGGGELYFDDILIRKNGYFVVLELEPLNP
>lcl|NZ_CP026520.1_prot_WP_042229579.1_1246 [locus_tag=PC41400_RS06535] [protein=glycosyltransferase] [protein_id=WP_042229579.1] [location=1446094..1447260] [gbkey=CDS]
MAHLFIAVQRVLGHVVPALAIGQQLKKKGHEVTLATHPANKELVVLAGLDFLPAGWGKFPEKFIYEQALEMDALLAGREV
DWMICDSSQAAPAYAAERRGIPWISLQTTVPYPEEALPGSRTVFNRLRQSYARELNGVRMRLGLPPLTDDRRTRGDLAGL
SPRLHLVNVHAALYPEEGGIPEQSVFTGSCTLQSGDRDRVNEASRLEPEAYPGRPVVLVCASSTERHDSREVMNRYIGAA
VSLSEENSFNLIVAEHKPYEGKRPLPPHVRWITAYPNHDLLLPRADAVITHGGCGTVQSVMRYGVPMVVIPLASDQHLLA
RTCERLGIALTVGPEEAGRGRIGQALHSLLAEGNPYMPRAKELAAIVGRDNPNEAAVRAIEQVLQNNS
>lcl|NZ_CP026520.1_prot_WP_042229575.1_1247 [locus_tag=PC41400_RS06540] [protein=phosphopantetheine-binding protein] [protein_id=WP_042229575.1] [location=1447273..1447506] [gbkey=CDS]
MLTGEAALEVIQKIGGLDSVAMDTNFLEKGIDSVKIVEILIEFEMMFNIDVLDDQLNLDELSTPKHIQDYVNGIIAK
>lcl|NZ_CP026520.1_prot_WP_084706583.1_1248 [locus_tag=PC41400_RS06545] [protein=phytanoyl-CoA dioxygenase family protein] [protein_id=WP_084706583.1] [location=1447606..1448397] [gbkey=CDS]
MSKEIMTGEMMEHYREKGFAVIPGVFSPDEAEEVKRALKEVWTRGIRSRTILQSLVFPLESLFPPLRNVHGSHEALRRFS
LDSRAFGTAARLLGENALLVGTTGFYKCPGAKQLPLHQDNYDIGAEPAAGCAFWVSLDGASPENGGLRMVPGSHRLGLLK
PRAPGSRSTYGQSVPVPDRHTEVDLTTKPGDVVVFNGYTVHGSHANRTRYGFRNSLVMHFVGESVRSLFVQNRELLNSAG
DNIFRPVNKSHSVKRLFLKKDSN
>lcl|NZ_CP026520.1_prot_WP_042229572.1_1249 [locus_tag=PC41400_RS06550] [protein=class I adenylate-forming enzyme family protein] [protein_id=WP_042229572.1] [location=1448414..1449847] [gbkey=CDS]
MDAKLFFRQLLDHSGLCAVDGTHEVSYSRITELAQSHARMLEIAGLVADPNRGQRTVAIDVSVGWRAIPLFLAALRQKVT
LVPVDPAKNPGLTENVLSWINPDLYITPELLNEDGELLPGLKPVTASPREELRDVAVVLYTSGTTGRPKGVMLTYGNIWS
NVGSILGYFKPAPEDRLYLFRPLTYASAFTGELLPSLFRGAAVYFRPPDTNPLSTMKSIQRDGITILGTTPTVAATFARF
RKRFDFGSLRCLMLSGECLTVRQAELIAEAFPGAAVWNAYGLTEASPRISCLTEVGAFLNRKGCVGKPLAGVEVKVLAEE
AKGRAAAEGEEGVLYVRGPNVMKGYYGDPAATSLKIKGCWLRTGDRAVIREGDIYILGRADSLIIRGGVNMDPSEIESEL
MKLRGIKEALVFADHSEDRTRVHAWITCEKGVPMTQIRSTIVHEIKDSRLWPDVIEMKEELPKTASGKLIRPIQSLR
>lcl|NZ_CP026520.1_prot_WP_042229570.1_1250 [locus_tag=PC41400_RS06555] [protein=aminotransferase class III-fold pyridoxal phosphate-dependent enzyme] [protein_id=WP_042229570.1] [location=1449852..1451177] [gbkey=CDS]
MHKSYPVVHPFTWMPPSGTLTTDTWLGQVTCLERGDGSWVYDTQGKAYLYTTTAVPTVGLSNWRVIQAMKEQMERLSFAS
TCAQTHGLVEPLADKLLRLCGSRYAKVFFTNDGSGAVETAMRLIRQHFISRGEAKRDKFISFDGSFHGTTYGSGSVTHLG
IREMFGRGLEDCLCAPAPNLYRPPGGVSSPEGAADFCLKELERLIEEAGPETIAAVLVEPIQAVNGVVLMPESFFSQVRT
LTRQYGIAVISDEVTTGLGRTGHWSLSEKYGLEPDVLAVSKGLTGGYFPMGAVLMSGELDDSLFGSGGILLHGSTQCGHP
VGCAAALEVLEIIERDGLPDHAAVRGQGILDDFADALADHPHVGDIRGMGLMLALEFVKDKATKEPVDFEWGRRLSGHLH
DEGILGNYFNGILLMYPPLNVSGEECDFLVHGVTRALRRMG
>lcl|NZ_CP026520.1_prot_WP_042229566.1_1251 [locus_tag=PC41400_RS06560] [protein=class I SAM-dependent methyltransferase] [protein_id=WP_042229566.1] [location=1451183..1451935] [gbkey=CDS]
MAAREHWHDFFGKDYLLFSEVILNEERTRFETGQLLSLLNLPEGASILDLGCGQGRIGVPLALRGYQVTGIDASDTLLDE
ARRRAQASGAEAVFKSLDMRELDFADEFDAVLNLGTAFGYLENEEENQDILHRVYRALKLGGQLIQETENRDFKLLYSAK
KTWDMMNGCPVWSRREFDSVTGRWNEEMTWFDNGELKSSRLNLRLYTAAELLRLHGNAGLDVKGIYGGFDLSPLTTSSPR
MLLHLEKKTD
>lcl|NZ_CP026520.1_prot_WP_009677210.1_1252 [locus_tag=PC41400_RS06565] [protein=cupin domain-containing protein] [protein_id=WP_009677210.1] [location=1451953..1452315] [gbkey=CDS]
METENKHKLIKWEEMNWEPGIYENTEMSYLYEDEQGRLCFLVKLHPGSFIPMHDHPRREFAYLIKGDLILNENEVMKEGD
FLTAGLAESHDVRTENGALLYVFIDYHVIRQRIVEIETTS
>lcl|NZ_CP026520.1_prot_WP_042229564.1_1253 [locus_tag=PC41400_RS06570] [protein=class I SAM-dependent methyltransferase] [protein_id=WP_042229564.1] [location=1452333..1453424] [gbkey=CDS]
MSGVTTVKTALFNETYHAAYEYVSDFTAFTLLNAMKAEGIDLNGPVRESRLLSQIACSEENRPLLNWALHFLEQQEFVRK
TTGPGGAVVLALTAKGRDWTNRGNPFPAVHVAPSLKLIEYAGTFWLRAIRGEVQPQHRLFNREGIALWEVYFHNAHDLYA
VHNQWAAEELADLIPAESCTILELGTGFGSATRALLTEIRLRSKFVEEYTVSDASRLLTAKAGRMLEGELPPHKLRQLTL
DINEVKSGGETYDFIYAVNVMHCAEDLALSLDALRRLLKPGGTIVLSECVREHYRSRLHQEFIFSLLPDFKPVGEADGNT
SCFGFLRPEDWWKVMQAAGFRDISVKMNEGPVRGAIITGSREG
>lcl|NZ_CP026520.1_prot_WP_042229560.1_1254 [locus_tag=PC41400_RS06575] [protein=helix-turn-helix transcriptional regulator] [protein_id=WP_042229560.1] [location=1453428..1454114] [gbkey=CDS]
MIVFWDREAGREADSDPLDSRDTPAAAYVADGLILHVGTAEDARSQAANGSCEALVVLTENERVEDGGLLADVSVPVYVF
TKADKQSRFVIQLLFHYLASIQSHVFSRKTFTGAGPDAEMDWERVLKHIQDNLGTGELSLESVAKECYVCKWQFSKVFKK
RFGITFRDYLIEQRIGEAKKLLVKGHSVTDACYAVGYGDMTHFGRIFRKKVGVSPSDYRKHYYSQFIL
>lcl|NZ_CP026520.1_prot_WP_042229558.1_1255 [locus_tag=PC41400_RS06580] [protein=hypothetical protein] [protein_id=WP_042229558.1] [location=1454160..1455011] [gbkey=CDS]
MRKRWQFLLAVLIVAGLLTEFYYYWNRPYKLPPVSDGMRLEEYNLAFEKEKAAAGGLAQGSAEELQKKVAAEPDNLAYGN
ALRVTMGKEGRIDAFVAFMKSLEQPPARAKLQLALAYVDQMQNESLGTASLGQISVHSIELMNEVLEKDPYDWLAHYARG
INNLYWPVGLQRIDKSIQDLSFCLAVTKKFEQDHPFYMWTLAYTALGDALVKKGEITDGMKIWKEGHQAHPDDAALKERA
QASKDQAVEIVVRERGMDQFQRPEPGISDLSPIWNPAKEGKGQ
>lcl|NZ_CP026520.1_prot_WP_042229554.1_1256 [locus_tag=PC41400_RS06585] [protein=beta-ketoacyl-[acyl-carrier-protein] synthase family protein] [protein_id=WP_042229554.1] [location=1455008..1456255] [gbkey=CDS]
MNHADFASGERSREGEQRAVVTGLGVVTSIGTDEVSFWRNLLRGETGVKPITVFDVEKHTNRMGCQIDDIPLSEYAPDAL
LKSGRATRILMPAIEQALRNAGLDKPVLSRCRVGLCVGTTMGDIDPLEDAIQGKRSEAFGGPHVITEQIGRLLGLSGPVW
TFTNACAAGNFAIARAMDEIASGRADVMIACGADSLSWVAFTGFSSLRAMAPDRCRPFDVRRKGLILGEGAGVLVIESQA
HAQARRIRPRAGLLGYGMSCDAHHITQPDPQGEGAVRAMRAALKMAAVPASAVDYVSLHGTGTQANDKMEAAALTAVFKE
TRLPAASSIKGHIGHTLGAASAIEAVMSVKCLEEGLAPPTLNLEQPDPACPVPVLMKEPAPGRYLIVLSNSYAFGGVNSS
IVLSAVHPERPENLE
>lcl|NZ_CP026520.1_prot_WP_084706581.1_1257 [locus_tag=PC41400_RS06590] [protein=beta-ketoacyl synthase N-terminal-like domain-containing protein] [protein_id=WP_084706581.1] [location=1456269..1457114] [gbkey=CDS]
MPQFIYLSGCGYYMAPSFAAIVERPDSRPSAGDSAEDSGDEALSSRIEDEPEEAAGGLKGLKRHSRAVRLACLAIREALQ
MASISGHAPGGKRNDIGVVVGTNDANLEAIDRLQQEAETYGVNNTNPGLFPETVLNVIAGQASIYFGLGGPNVTISGSDF
SSVKSLQYAADMLESGSASHVAVCIVRLQPPAVFEGSIRYRAARESVVALVLSRQPSGGSGSGLKLRLESVTDLPDAAAT
RVTDEQLPAVLAVKLRSGELPSGTSVIYADAGPEGLLAVEF
>lcl|NZ_CP026520.1_prot_WP_042229551.1_1258 [locus_tag=PC41400_RS06595] [protein=beta-ketoacyl synthase N-terminal-like domain-containing protein] [protein_id=WP_042229551.1] [location=1457151..1458419] [gbkey=CDS]
MKNTSVLVTGTNMLCSLGMTNHEVWQAALSGSVPEPSRRSYPFPDGTRLEYPVFSLPDFDLSDWLEEKSVRWLAENHLLD
DRDFCMLLLSIRQALEDASFRITGGSRIGLVIGHENLGAVKLTDKFIRYQDATPQTDQAEKMPLFNEFRREFFNIQTFPY
LFYLAQLLGVNGPGYVVNNACASGLYALELGRMLIASGSADLVIVACSDYAHASEYLWLQEKGFVSGQMKIRPFDLHRDG
SILGDGAGTIILESAEHAAGRPVEPLAVYRGGAFLQEHWHMTLPDVSRHSYASVITEAAGHGTAPEIDVIVPHGSGSTLW
DRYEANEIHRAFGRLNRPVPAVTSFKGYFGHLLGASAMIETVLLLHCMKREILLRTLNYEQADPRVQLPVQMQTARKKTV
SALKSVPAYGGFHAAVLFNKPD
>lcl|NZ_CP026520.1_prot_WP_042229548.1_1259 [locus_tag=PC41400_RS06600] [protein=SDR family NAD(P)-dependent oxidoreductase] [protein_id=WP_042229548.1] [location=1458428..1459177] [gbkey=CDS]
MATTTYSFAGKTILVTGGSRGIGRAVVKQLASAGARVVFTYSKDREAAETLLDETGNDSSRVSCVQADFTDRLAVAKMVA
NLLEEGPVHGVVNNAGILRDTPMYRMADEQWDTVLQVNLTSLHIILKNLIRPLAASRGSVVNLSSVTGIAGAAGQTNYAA
SKAGVIGLTKALAKEAGPLGIRVNAVAPGFIGTDMLESIPARRKRDLEKDIPLRRLGTPEEAASAVLFLLSEEASYINGA
VLVVDGGLL
>lcl|NZ_CP026520.1_prot_WP_042229547.1_1260 [locus_tag=PC41400_RS06605] [protein=acyl carrier protein] [protein_id=WP_042229547.1] [location=1459192..1459455] [gbkey=CDS]
MNVKAVKSRIIEVTVSLLDSTEPVEELDADAFLRKPLPEIGIDSLAVLELVVTLEREFGVRMTEDDLGGIATLEDILTFI
TGRAGQS
>lcl|NZ_CP026520.1_prot_WP_042229544.1_1261 [locus_tag=PC41400_RS06610] [protein=CRTAC1 family protein] [protein_id=WP_042229544.1] [location=1459496..1461226] [gbkey=CDS]
MQVKPLSKMLLAATALLLTATGCGAGQSRTDFGFSLKETTQASGVKFIHSKPTFDSKVANIMPWMASTGAGVFTADYDGD
GFMDLYFINSAKGSKNALFHNNGDGTFTETAESAGLADVNTDGISEAAVWFDYDNSGYPSLFVGSWGKSRLFHNNGNGTF
TEVTDQAGVGYKGYVNKAIVLDYNKDGNLDLYLSCYFHEKNDLWNLTTTKIMHNDFERARNGGRNVLYRNNGDGTFTDVT
AELGVGDTGWTLASGTWDVNNDGWPDIYNANDFGPDTLYLNEQGKKFTAVVQPRGIGDDTFKGMNVDFADIFHDGKPAMY
VSNVSKSQYILEGNQLWHEKDGQFIDRGKEMGVNLAGFSWGARFMDLDNSGNSSLVVQTGFVSASKKKDYWFDLGTLATT
PGNIVEDTKNWPAFVDKSLSGYENKFLFYPQGNKFTDIAEQVGLDFVEDGRGISAIDIRNQGKLDLVFANQGGPARVYEN
TVTNQNNWIKLDLIGTAPSSRDAIGARVTFEVNGVETVMEKDGANSFGGQSDPRLHFGLGKADKADKITVRWPSGRVEQF
RDVPKNQILRLTEKKD
>lcl|NZ_CP026520.1_prot_WP_042229543.1_1262 [locus_tag=PC41400_RS06615] [protein=RnfABCDGE type electron transport complex subunit D] [protein_id=WP_042229543.1] [location=1461227..1462012] [gbkey=CDS]
MSKDPRLYILAFLLSFVAAGQLFLGFFQRWDSFFASVFTAVVLEFVLVGLLKKKWAFPLSALITGIGISLLLSSYAVWPY
VLTAVLAVTIKQFVRLRGAHIFNPNNVAMVIMLIALPAYAVSTPKQWTNGIEVMSLIMILGLFAAYKAKRLDSVLAFIGG
FSFFAAIRVGFFGEPALYAFGPMMGASFQLFSFFMITDPKTTPPTRKARILFAFLIAFTDAWLRVNQITNSQFYASFLVT
LLLGVPYRLLAKKAGETGIPV
>lcl|NZ_CP026520.1_prot_WP_042229541.1_1263 [locus_tag=PC41400_RS06620] [protein=hypothetical protein] [protein_id=WP_042229541.1] [location=complement(1462157..1462357)] [gbkey=CDS]
MPSLDRFDAGLPDRQAEEPSQVTECAFDRCRSPIYAGEKNWDFDRDWFCSAACIARHLGAEKRYVE
>lcl|NZ_CP026520.1_prot_WP_042229540.1_1264 [gene=lexA] [locus_tag=PC41400_RS06625] [protein=transcriptional repressor LexA] [protein_id=WP_042229540.1] [location=complement(1462465..1463304)] [gbkey=CDS]
MNEINQRIKELRLKHNLTMAKMAAMIGVSPGNISDWENGKKKSTPTARALISISKQFNVSLDWLMTGTIPDFPANFVPFP
PIEKSHPYIDALYDVAGQLEDEDFVLLQTMAAHLANKNRAQTPGDLDLYRSGKVRAASIQSGSELREASLSAEYYVRLPL
IGKVTAGSPILAAENIEEYFSVPKNLVGQGKHFILRIQGESMINKGIENGDLVLVRQQLSADNGEVIVALIDGEEATVKT
FFKESNRVRLQPANDRFQPLYPANVSILGKVVSVLRDAE
>lcl|NZ_CP026520.1_prot_WP_241688807.1_1265 [locus_tag=PC41400_RS06630] [protein=helix-turn-helix transcriptional regulator] [protein_id=WP_241688807.1] [location=1463686..1464333] [gbkey=CDS]
MNEVDLLVFVHKAEKCKDYLADLLLLSKKVKLTEFVDFNSALQLVKQAHYKLIVFLTFEPAFPRTWNVNELRINSPLLAL
AGADKISLSSLISLIDMVAQNEEVSHDNRLFAESLAYIDENICENDLNLEKVASQIYVSRCHYSRMFQKYVGTGFKEYVM
NKRIQKAKLLLQRGEPVTDVCFSVGYNDLTHFGRIFKRKVGVNPSVYRLNKAASQ
>lcl|NZ_CP026520.1_prot_WP_042229538.1_1266 [locus_tag=PC41400_RS06635] [protein=DUF1702 family protein] [protein_id=WP_042229538.1] [location=1464431..1465435] [gbkey=CDS]
MQVIAVVSAIGAFLAVLAPFIYLKAFHRSIPKMVAKFPLTHNVHYDRKFPQILRAFLSGSNWALKPVLPMSGLRARLDNR
FEPFYKGFAYEGAGMGLGVKSALFPSRGRRFEAYIHQLSPGYLYQYYVGLGWYLTIRHGFRYGGYKRWIKQLHPRYAPIV
FDGAGFKTALFHYKKNKQIIQKFRHFPFSYQRVCYQGFGRCLWFLTEFDLQKTAAELENLPAEFRHDAYSGVGLAAAYSF
FDQLAFAAHADKQIPEQYRPAFRQGLAFGWEARRLQDPVYWDEQLERYTYPVILKVTGFVDRVHEAVQLLGEDDTEADYY
MRWMDTVRSLLKEK
>lcl|NZ_CP026520.1_prot_WP_042229537.1_1267 [locus_tag=PC41400_RS06640] [protein=YqaJ viral recombinase family protein] [protein_id=WP_042229537.1] [location=1465479..1466375] [gbkey=CDS]
MRAKKLVNTRAVSDGEWLAWRRQGIGGSDAGAVCGADKWKTALEVYLDKTGELPPVEEKEAMRLGSKLKDFIAREFTDRT
GHRVMRRNWIFQHPEHSFMLAGLDRWVIGSHAGLLCKSTGEFSKEEWEDGRVPAKIDFQCQHYMAVTGAEHWWVAVLIGG
NKFRTVRLERSEERIKRLIDREAVFWNEHVLRRRPPAFDGTDASSELLNRLYPPATSRNSSVPLTEEAERWIGQYRQGAE
EERAAALRKTEAENKLKGLLGDHEQGTVGNYKVEWKSINRKSARFAPYRRFQISTISK
>lcl|NZ_CP026520.1_prot_WP_042229533.1_1268 [locus_tag=PC41400_RS06645] [protein=RnfABCDGE type electron transport complex subunit D] [protein_id=WP_042229533.1] [location=1466388..1467236] [gbkey=CDS]
MRNEMNTHLRSVVSWRDYRIDPRYFILVFLSSFAIAGQVYLGFFQKWDAVLAAVVCCTGTEIILNRLKNKTWMFPLSALI
TGIGISLLLSSNQLWPYALTAVIAIALKYAIRFKGGHIFNPNNVAMVIMLYSLPQYAVSTPKQWTNGIGVMILILCLGIV
VCYIANRLDAVLAFLGSFTLFAFVRHFFFGAPLFAALGPLMGASLQLFAFFMITDPKSTPATRSARIAFAFFVALLDAVF
RINRIPNPQFYALFLVCLALIIPYRLWASRKTEPEPGLTANS
>lcl|NZ_CP026520.1_prot_WP_241688808.1_1269 [locus_tag=PC41400_RS06650] [protein=hypothetical protein] [protein_id=WP_241688808.1] [location=1467267..1467650] [gbkey=CDS]
MTKKDDRAINSTKEVNAMCRNGTCRGAFNGGEGTGCGRHCANLRTEANFNEETDDTIAKPVSRAANQVEKQPENRMPSRT
ETQPQHRERNDELKQEMRDRLPADHLPSRYFLRTKYFKYWDFWDHFR
>lcl|NZ_CP026520.1_prot_WP_042229532.1_1270 [locus_tag=PC41400_RS06655] [protein=bifunctional rhamnulose-1-phosphate aldolase/short-chain dehydrogenase] [protein_id=WP_042229532.1] [location=1467892..1469961] [gbkey=CDS]
MTANLWDAEIAVGVKKGLDELVYRSNLLGFDRTVANWGGGNTSAKTAVTDFRGREIEVMWVKGSGSDLASMKAENFTGLR
LDDIRPLMEREEMTDEEMVAYLGHCMIDGRHPRASIETLLHAFLPFKHVDHTHPDAIIALCCADNGKELAREIYGDRFVW
VPYVRPGFTLSKMIAEGVRGNPAAELVLMEKHGLVTWGETSAACYKKTIEIISEAENFIESRVNRAELFGGAKYETLADS
ARREILAQVLPVIRGAVGGERKMLLTADAGEDVLAFVNSREAQALSQVGAACPDHLVHTKMKPLYVDWNPAEADVPALIE
ALKSGIAAYKEAYGAYFDRNRSEGDVMFEAAPRVILIPGLGMVNTGKSRAMAQVSGALYHRAIAVMRGATALGNFVSLSE
NESYNVEYWPLELYKLTLAPAEAEFSRKVAFITGGAGGIGGAAARELLKGGAHVVLADLNLEGAQTLAAELNRQYGQGRV
IAAAMDVTSEEQVGEAYKETVLAFGGVDIIVNNAGLATSSPFEDTTLKEWNLNMNVLGTGYFLVAREAFKIMKVQGTGGS
MVFVGSKNSVYAGKNASAYSTAKAAEIHLARCIAAEGGAYGIRVNSVLPDAVLQGSAIWNSGWRNERAAAYGIEPDQLEE
YYRRRTTLLVNVYPRDIAEAIAYFASSKSEKTTGCMLTVDGGVPAAFTR
>lcl|NZ_CP026520.1_prot_WP_042229530.1_1271 [locus_tag=PC41400_RS06660] [protein=DeoR/GlpR family DNA-binding transcription regulator] [protein_id=WP_042229530.1] [location=1470128..1470889] [gbkey=CDS]
MLVAERWQKILALVNERGSIRVTELSELCGVTEETIRRDLDKLESEGRLMRSHGGAVRVGESQPEIPFKERETVHADLKR
SIAREAVKHIRENDRIILDASTTAWYMAQILPDEPLTVLTNSIQVAVELSSKEKIQVVSTGGRLLPQSLSYAGPLAERSL
DTYHVDKAFLSCKGVHTARGISESSELQALVKAKMVAISDEVFLLADHSKFGQQSFAHVASWERIGHVITDSSTSADSLE
ALREKGVKVTRLT
>lcl|NZ_CP026520.1_prot_WP_042229527.1_1272 [locus_tag=PC41400_RS06665] [protein=(Fe-S)-binding protein] [protein_id=WP_042229527.1] [location=1471027..1471770] [gbkey=CDS]
MKVSLFITCLSDLVYPRVGQAMVRLLARYGIQVHFPEVQTCCGQPAFNSGYWDDARKSARTLLEAFDDSDFVLSPSGSCT
GMVQHYYPKLFQDDPEMLAKAEALREKTYEFTQFLVGVLGVTDLGAVFPHRVTYHPSCHGTRLLGIKDEPASLLEHVRGL
ELVPLPHAEDCCGFGGTFAVKMSDISGAMVAEKSDHVLETEAEVLVGLDMGCLMNISGNLLYRGRPVKVMHLVELLEEGV
RRADEHV
>lcl|NZ_CP026520.1_prot_WP_042229524.1_1273 [locus_tag=PC41400_RS06670] [protein=LutB/LldF family L-lactate oxidation iron-sulfur protein] [protein_id=WP_042229524.1] [location=1471757..1473268] [gbkey=CDS]
MSMSSGRVSVKDRAKIALNNDFLRKAVKFTTERLKTGKLNAAAEQGNWEEWRERGRQIRLHTIAHLDYYLGAFVENARAQ
GVHVHFAETAEEAVSITMEIARHRKAKSVVKSKSMVSEEIHINRALEELGIEAVETDLGEYIIQLAGEAPSHIIIPAIHK
NRYQIAELLSEEAGETLPPDTAVLAGFVRRKLREKFLEADIGMTGCNFAVAETGSIVLFENEGNGRMVSTVPKTQITFMG
MERIIPTLDDLEVMATLLPRSATGQKLTVYMSVISGPRREPDGDGPEEMHVIILDNGRSLQLGDPDFQELLNCIRCGACL
NACPVYRHIGGHAYGGTYSGPIGAVLTPALNRNTAEWDDIAGASTLCGACYEACPVKIPLHDMLVYLRGRKVEKGHGDAA
EKAGMSAFGYIMADAKRLRRVAALGRFGQRFVARGGVIKARIGPLKGWNAHRYAPALPPKTFREDWKSFAEEAAGSLPEM
DAEVKSRMERILDARRKGGGRHE
>lcl|NZ_CP026520.1_prot_WP_042229522.1_1274 [locus_tag=PC41400_RS06675] [protein=LUD domain-containing protein] [protein_id=WP_042229522.1] [location=1473261..1474013] [gbkey=CDS]
MSKSAADFLKGLEAESRRKQEAFLGGIASRLGRPRITEPPARPFRGAPDFWNAVDLDEEARIRLFMDNWRKAGGHAEQLA
SMEEAAAFVRQLIRDLEARRVIRQDQPELEAIASVLETVEGLDRLTVWNAEHASDEEKEAMLADAAGADIGLVAADCAAA
HTGTVVLMSSPRKGRSVSLLPAVLVAVVPASRLKMRLGEVLRPLDGLAAGARPAGIHFVSGPSRSADIENDLTIGVHGPG
IVYALVVGDC
>lcl|NZ_CP026520.1_prot_WP_042229519.1_1275 [locus_tag=PC41400_RS06680] [protein=M15 family metallopeptidase] [protein_id=WP_042229519.1] [location=complement(1474293..1475141)] [gbkey=CDS]
MTPKKRITRLLILASAVALFAGCESSKEAGGNSPSATSTPGTSAVTSPSANPSGTPGSTSKPTTSPTPNPSSKETGAGKG
TDGSEQVAAKPESMTVLVNKQFKLPDSYKPSDLVYPDVNFVFAEKIEKRMMRKEAAGALEKMFAAAKNDGAPLAGVSAYR
SHSTQTTLFNRYVQKDGEAAAKKYSAVPGHSEHETGLAIDVSGSTGKCAAEDCFGGTPEAKWLEDHAHEYGFIIRYPKGK
ESITGYQYEPWHLRYVGTQMAKDIKTKNTTMEEYLNAVPVAK
>lcl|NZ_CP026520.1_prot_WP_042229517.1_1276 [locus_tag=PC41400_RS06685] [protein=hypothetical protein] [protein_id=WP_042229517.1] [location=1475959..1477029] [gbkey=CDS]
MNKKIWTSVLALLLTFTVVLAGCGKKESPKTALEESYTAASTMNSADFNMNMTLNLDFPESALKADPSLEMGANLLKNLD
VTAKGVYQADPMQLEMTLEANLKGDMPMKISLPMVMTKDKMYVKVPNIPLLPIPKDAVGKYIVVDVNELNQTTEGKVNVN
AIDVKKQQEFGQEAMKIFTTNFDEKTYFVEVDKKNAGLPESVDAKQVVKFQITNENLEPAIKTIVEKVAPQLLDLVSSDK
YKDMLQVDPAEIEKAKKELAAGSSKEDTQKAIDELKKVLKINDFNILTAINKDKFPAYQEINANLDITEEKQGTIKIGTK
IKTEYSNINKKANLQPLPTDTLTIEQFSQKMQASGM
>lcl|NZ_CP026520.1_prot_WP_042229514.1_1277 [locus_tag=PC41400_RS06690] [protein=hypothetical protein] [protein_id=WP_042229514.1] [location=complement(1477164..1477412)] [gbkey=CDS]
MQYDWSSISRTVGFYRRHQVIPSIEEIEDTWQRARDGSREAAIRFLDWRRHCLLAKTDEQRAAMILVMNYSHKLFNRDST
SE
>lcl|NZ_CP026520.1_prot_WP_042229510.1_1278 [locus_tag=PC41400_RS06695] [protein=sugar phosphate isomerase/epimerase family protein] [protein_id=WP_042229510.1] [location=1477796..1478653] [gbkey=CDS]
MKIGLSSYSLYRAIKAGELDYVSALDWIAANGGEHMEVVPLEIDFTREPQLVQNIKEKAKSVGIELSNYAVGANFLTQNE
GAYEEEISRVKREVDVAAELGVKLMRHDVARHEDVSIRTFNANLDRMIEACRTIADYAAQYGITTSIENHGYLVQASDRV
LTVVQGVNRPNFKTTLDIGNFMCADENPVASVQKNLPFASIVHIKDFYLRPSYRNPGEGWFKTTAGNYLRGAIAGQGDID
MWEVLRLVKRSGFDGYLSIEFEGMEECKQGSKIAMDNVKRIWAEV
>lcl|NZ_CP026520.1_prot_WP_042229507.1_1279 [gene=thyA] [locus_tag=PC41400_RS06700] [protein=thymidylate synthase] [protein_id=WP_042229507.1] [location=1478878..1479672] [gbkey=CDS]
MNKYLELLKDIKENGTRKEDRTGTGTISVFGRQMRMNLAEGFPLLTTKKLHMRSIIHELLWFLKGETNIHYLHDNGVTIW
DEWADEDGNLGRVYGAQWRTWQAPDGRAIDQITQVIHQIKTNPDSRRHLVSAWNVAEVDQMALPPCHYAFQFYVADGKLS
CMFQMRSVDTFLGLPFNLASYALLTHMVAQQCDLEVGDLVWTGGDVHIYLNHLEQVEKQLTREPFPLPKLEIKRKPDSIF
DYAYEDFEFVGYQSHPGIKAPIAI
>lcl|NZ_CP026520.1_prot_WP_042229506.1_1280 [locus_tag=PC41400_RS06705] [protein=dihydrofolate reductase] [protein_id=WP_042229506.1] [location=1479783..1480268] [gbkey=CDS]
MSISFIFAMDRNNAIGIDNKLPWHLPADLKYFKKMTTGHPILMGRKTYDSIGRPLPNRDNIVLTRQSGFAAEGCTVVHTV
EEVLERYTEGEVFVIGGTEVFLLFWPHVDKLYVTYIDDTFEADTFFPEIDPGEWTLVSDEPGIRDEKNPYDYTFRVYERA
Q
>lcl|NZ_CP026520.1_prot_WP_042229504.1_1281 [locus_tag=PC41400_RS06710] [protein=Gfo/Idh/MocA family oxidoreductase] [protein_id=WP_042229504.1] [location=1480524..1481432] [gbkey=CDS]
MQSSSLRIGMIGLDTSHAVEFTKLLNDPEHPWHVPGGRVVIAYPGGSEDFELSRSRVRGFTEKLRGEYGVEIADSPAAVA
AACDAILLESADGRVHAEQFRQITAYGKPVFVDKPLAVSSRDAGEIAALAVRHGAALLSSSSLRFAEGLTEALSDSEKGA
VTGADCFGPLALEPTQPGLFWYGIHMVEMLYAALGPGCVHVTASAGESHEVVVGVWADGRIGTVRGSRAGSPAFGAVVHR
EGGSRAVDISADAKPFYASLLERVMELFRGGEPAVDMRETLEVIRFIEAANESRETGRRVRL
>lcl|NZ_CP026520.1_prot_WP_042229501.1_1282 [locus_tag=PC41400_RS06715] [protein=YcnI family protein] [protein_id=WP_042229501.1] [location=complement(1481790..1482434)] [gbkey=CDS]
MNKLKMQAAAVVAGLMLFAGVASAHVTVNPAEATAGSYQKFTVRVPTEKDVPTTKIEVKFPVDAVSISRFEPKPGWKYEI
AKDGADKITGVTWTAENGGLSKTEFGEFNMQGKVADNATSIVWKAYQTYQDGSVVEWTGAEGSDKPASVTKVAAKPSGAG
TDSHGNTSTAPGSATAAPAQQAGASASNTPLYLSIAAVVLSAASLLLSLFRKRA
>lcl|NZ_CP026520.1_prot_WP_042229500.1_1283 [locus_tag=PC41400_RS06720] [protein=response regulator transcription factor] [protein_id=WP_042229500.1] [location=complement(1482528..1483196)] [gbkey=CDS]
MSETCYRVLIADDHPLARQAIQSLLEDDSSFNVVGEAYNGEEAIALCGNLQPDVVLMDIHMSPVNGLEATRQIKQRYPHI
RIVILTVSDDVADLFTAVQFGAQGYLLKNMDPDEWLNYLHALLDDNSQVARQMADRLMRRFRPSGASAASGQPLPQLTAR
EHEIVLYVAEGYNNRQIAEKLLISENTVKNHMKNIFEKLGMDNRVQLTAYAVKHGMSRKDHN
>lcl|NZ_CP026520.1_prot_WP_042229496.1_1284 [locus_tag=PC41400_RS06725] [protein=sensor histidine kinase] [protein_id=WP_042229496.1] [location=complement(1483159..1483995)] [gbkey=CDS]
MSLKTLKWLAALTPALFIGGFEFIRHEFMLSYMSMETGNWYITFLTLLVSYIFATWMFRSIERTNNRLARETSKRAVYEE
RERLARELHDNLAQTLFFINVKLKQGKTEEAKAAVSEIDHHLRQAIFNLRSLPEEGTSFTLRIGKWLQDWEVLTGIETVQ
NLRATDKVLTAAEEVQIFAIIQEAFTNIRKHSRATQASIDLETSGPEWSLRIADNGVGFAEDESQRHRYGLTMMRKRAAE
LGASWELQSSPENGTELSVTSSRRKPHVRDLLSRSHRG
>lcl|NZ_CP026520.1_prot_WP_042229494.1_1285 [locus_tag=PC41400_RS06730] [protein=hypothetical protein] [protein_id=WP_042229494.1] [location=complement(1484221..1484466)] [gbkey=CDS]
MFDILIYQKQKVKVSCGKDTITGIISNHYEQFGLIKMDNVLLPIEHILMIEVVEEEGDPFCYADLPAQPINKGASLREML
Y
>lcl|NZ_CP026520.1_prot_WP_042229493.1_1286 [locus_tag=PC41400_RS06735] [protein=hypothetical protein] [protein_id=WP_042229493.1] [location=1484697..1486037] [gbkey=CDS]
MEIKRPPSRDEIVIGLIGPEELLDKAEETIKAFPSFHPLSLKWEGEPHRLPDVQRLGEQAEVLLFTDPDTFRLAKQSVTF
RIPAHYVPLTGTGLYRALYLAGHTGSPGKLTVDGLTEPVVAKALRELGVNGMDVVCYGGQIDGSLQELIDFHRAGASGDT
VALTGMKAVAEALAKEGLRSEWIAPTEHDMTISLQRALLSTETRRSKEGQIVIGFINIDGFERLAARSGSEHEVQRMKLD
LHRMVLDYVEHLDGYLNTLGGDEFLFMTTRGIFERETGGYKSIPLAKDLSLMHDLTLSIGIGFGRSANEAGTHARLALRR
SKDAGGNITFIVREDRSVIGPLEMTEPNRFDLSLLDAELIKEAGRAGMTPAYLSRLVAYVTRFGKLDYTAQELAGVLDVT
VRSTHRFLLAWQDAGLVDVIGEERGVSKGRPKQIFRLAFLQERIRS
>lcl|NZ_CP026520.1_prot_WP_042229490.1_1287 [locus_tag=PC41400_RS06740] [protein=VTT domain-containing protein] [protein_id=WP_042229490.1] [location=complement(1486308..1486901)] [gbkey=CDS]
MTWESIKNVLTEENIAQLLSHYNSLGPLPGILLTFIEAFIPVLPLFVIVMGNAASYGLGFGFLYSWLGTSLGAICLFLLF
RRLSNTAFMQKLTRSPKLRNGMNWIHNHGFGTIFLLSCFPFTPSSLVVVAAALSRVEKMIFLPAIVAGKAVMVFTMAFVG
YDIMSFIRQPWKFLIVLVVIVLLWLLGKKFETRMHRA
>lcl|NZ_CP026520.1_prot_WP_042229944.1_1288 [locus_tag=PC41400_RS06745] [protein=type II toxin-antitoxin system PemK/MazF family toxin] [protein_id=WP_042229944.1] [location=1487148..1487729] [gbkey=CDS]
MKTDLTRIQDYLEWLKTKIYLDSLSDKAKTRALRRGEVYKCNLGKGIGSEEEKERPCIILQNDVGNKHSGNTIIAPITHT
AGIPSVSVELKGNYTYLEVGKEKQLTGYVLLANIMTVSKSRITGSCLANIRGEMDEIEEKVMVSLGMFKKYKDLLDKKER
DKAFIKRLLQENYALKEKLDLYERPDSASEIHG
>lcl|NZ_CP026520.1_prot_WP_042229488.1_1289 [gene=pxpB] [locus_tag=PC41400_RS06750] [protein=5-oxoprolinase subunit PxpB] [protein_id=WP_042229488.1] [location=1487944..1488765] [gbkey=CDS]
MDPKSSSAIVFEPLGDQAVVLRLGEGISEEVHRRVLEAAEAISRSPFPGMVEIVPTYTTVTVWYSVVEAAVAAGRHFSPS
ELEETEAMLAGGEPVVYTWVRRHLERVLETGTAENPVLPASPSREIFIPVCYGGSFGPDLEDAARALGMSPQQVVRLHSG
GVYRVYMIGFVPGFPYLGGLPSSLALARRSNPRVAIPAGSVGIGGSQTGIYPLPSPGGWHIIGRTPLDMLRTSENPPVLL
QAGDTVRFVPVDEGDYERLCREEAGTAKEGNRR
>lcl|NZ_CP026520.1_prot_WP_042229487.1_1290 [locus_tag=PC41400_RS06755] [protein=biotin-dependent carboxyltransferase family protein] [protein_id=WP_042229487.1] [location=1488762..1489778] [gbkey=CDS]
MSLLVENPGLYTTVQDTGRFGFRRFGFVSGGAMDTLSMQAANILAGNPRHAAVLEWTLQGPRLLLRQDTLAAWCGPELAV
YAGGTPVPGWRPVYIRAGTTLTLGSGRNGCRGYLAIAGGFDVPLALGSRSTYTRAGLGGLEGRPLQAGDELLPGRPSPAA
ERMTAGLRLQASAEDNDVRAAAWEASWAVRPGGTGGNPAVLRVVRGSETGRFKEGLDALTASVYRVQPQSDRMGCRLQGP
ALELSVPGELVSESTAPGTIQVPPDGQPIVLAADSQTTGGYPRIGHVISADLPVLGQLAPGASVRFAEVTLHEAQAELMR
QEMNLRLLQTGVRARMGR
>lcl|NZ_CP026520.1_prot_WP_042229485.1_1291 [locus_tag=PC41400_RS06760] [protein=5-oxoprolinase subunit PxpA] [protein_id=WP_042229485.1] [location=1489782..1490549] [gbkey=CDS]
MASIDLNCDMGESYGAYRFGRDEELLESISSANIACGFHAGDPGVMRRTVALCLEKGVAVGAHPGLPDLAGFGRREMAVG
AEEVYELTLYQLGALQAFVQAEGGTLRHVKPHGALYHMASARGDLADAVARAALRVSGGALVLLGPPGSELLRAGKAHGL
RVAAEAFADRTYRPDGTLTPRTEPGALLSDPAQAAAQVLRLVREGAVTSARGETVPLTAETVCVHGDGPHAASIARSVRA
ALDEAGIAVCPPHGA
>lcl|NZ_CP026520.1_prot_WP_009676152.1_1292 [locus_tag=PC41400_RS06765] [protein=CD3324 family protein] [protein_id=WP_009676152.1] [location=1491248..1491544] [gbkey=CDS]
MSYKNGKDILPASLLKELQNYIQGEIIYVPKKEKKRAGWGENNGTRLIIERRNREIFRLYQNGSTVLELIQTFHLSEDSI
RKIIGKTRERLKEAAATP
>lcl|NZ_CP026520.1_prot_WP_042229484.1_1293 [locus_tag=PC41400_RS06770] [protein=response regulator] [protein_id=WP_042229484.1] [location=1491739..1492122] [gbkey=CDS]
MDLNPYRILLADDEAALRFLLTETLSDEGYAITEVMDGKEAVAELVRERYDLVILDYMMPEKTGVEVCEWLRSEGGINRE
VPVILLTAKALDKDREKATAAGVSTYIVKPFSPLQLVDTVEALISGA
>lcl|NZ_CP026520.1_prot_WP_042229480.1_1294 [locus_tag=PC41400_RS06775] [protein=diguanylate cyclase] [protein_id=WP_042229480.1] [location=1492161..1493894] [gbkey=CDS]
MNQPSSRGHGADAEGRDKRLLKQGRRLFIKELAHQIAQMDDLLVIIASSPEAKDLQDMYRIVHMLKGSAPMFQFGRIGKI
AEELVRVWEWTQSLHPSAALTADFTPLVNESVTHTSQYIRHLLMEHDVSVIELEVDEQKENGIVQTMGGKRRLLIIDDDD
VLRTYLVRRLELDGYKVDEASDVPSAKLLLRQQAYDLITLDLMMYPQSGYELFDFLKEDPTLKWVPLIVLSGRDDLSDKV
RCFHLGADDYVTKPFQYEELAARIYGIVKRTKNFEELAFRDPLTAVFNRRYFDHQIQLELERVQRYPSAISLAFIDIDNF
KRINDRHGHHTGDLVLQGLAHLLQTRMRSTDLVFRFGGEEFVVVMPGTGAEAAAESLRALLGMAHLQPIVSDEGEDYWIT
FSAGVSAWHPDLSIDEWILKADQAMYRAKQDGKDRIYIEEETEAAPAGSEGSLRHRPKLLVVDDDKILRAIVVSGLQHLS
LDIEEAADGEAAYERIREGGIDLCILDGIMPKMSGIELLERLNAEPPVSGTMCKVIMLSGRKKEESMDTLQKLGVHEYMT
KPFSMVELEMNVKRLLG
>lcl|NZ_CP026520.1_prot_WP_042229479.1_1295 [locus_tag=PC41400_RS06780] [protein=PhzF family phenazine biosynthesis protein] [protein_id=WP_042229479.1] [location=complement(1494059..1494934)] [gbkey=CDS]
MSLSKQDFYIVDVFAERKYAGNPLAVVICPEPLPAPDMQSIAREMNYSETTFLQPEPDADGSYRVRIFTPNSELPFAGHP
CLGTAWIIRHLYSPDAQEITLKVPAGNIPVTFGSDGVLWMEQLQPTFGSLVSASAAAAVIGLETEEIDERYPAQVVSTGI
PFLLIPVKTLASVRSAVYRSEAAAALPAEAAVPVMPFCPETYDKANGINARVFVDELGVPEDPATGSAGGCLAAYLARCR
YLGGPSVDTVVEQGYEIGRPSLLYLRASEIDGRFAIHVGGRVFFTAQGQLV
>lcl|NZ_CP026520.1_prot_WP_042229476.1_1296 [locus_tag=PC41400_RS06785] [protein=DUF2203 domain-containing protein] [protein_id=WP_042229476.1] [location=1495165..1495572] [gbkey=CDS]
MAKRYFTVAEANALLPSVQDQISGLKEIKKKFEDKYAELREKRQAVHKLGGTEPNDPLFELECDLEFLQIEARGMIQHFY
HQGIELKDIDSGLVDFPSILGGEEVLLCWRSGEERIGYYHSRHDGYTGRKPLPGD
>lcl|NZ_CP026520.1_prot_WP_042229473.1_1297 [locus_tag=PC41400_RS06790] [protein=copper resistance protein CopC] [protein_id=WP_042229473.1] [location=1495623..1497284] [gbkey=CDS]
MKEKVVKPARRAFIAAGLLCLLVILGGLFPREAFAHAILTGTDPAPNSTLAQAPEQIKLTFNEPLEEGVYYIRVLDQSGA
KVTDNAAVMTKDRTGIVLQLPKLDDGLYVVSYHVISGDGHPVGGSYPLTVGKAAPAAGSGATGGTGGTTAGHNHNLGSSG
VDGLIQYAARGLWYAAMLALTGWLLWRRSKSGRGLPQEVSARWTLNLQRFYLVALLFVILTHVEALLGDGGLSQLGSLLT
GTGIGISWLVSLGLSLTGFVLLGRAAWLDYAWAVALLAVKALSGHAVTFPPVLASVAADFIHLAAAAVWVGGLTAAAAVY
RNRREEFTSFLGRFSGAALGSIVALTVSGVVLTLLFLPSLDMIIYTQWGILMLIKVGVVVLVVGTAAFLRYTMKNKRDKA
TAKLYKVDFSLMLIILAVVGVLTYLSPAPPNEPLNWHEMGKTVHMTAEVSPKAPGTNTFKVTVWVLDTMGPPKHVQMNAQ
YEPKEEEEATLAPIQIPLELQENQEENWLEGYNMYTYYTQGPYLPLPGKCTLEVRIMDSNDDEAVYKKEFRLY
>lcl|NZ_CP026520.1_prot_WP_042229472.1_1298 [locus_tag=PC41400_RS06795] [protein=DUF418 domain-containing protein] [protein_id=WP_042229472.1] [location=complement(1497430..1498623)] [gbkey=CDS]
MIGPSTGNERIIALDIIRGFALLGIFLVNMPTFSGSEFAVYHGADRWLRLLFDLFVQGKFYLIFSFLFGLGFYIFMSRAE
QKGFGPGLFRRRLLALLLIGFLHLALFWRGDILTLYAITGFLLPVFYRMKTETVRKWGLLLALSYLLLILGLPALIVALV
AYAKGGIDPSLGGPLYTEAEAAASLHAFRTLSWPDWLAWHWRSEVWDSLSSQLSMLPVVLGMFLLGLAAGRSGLLHDVSA
HRERWRKIRSRGLALSLPLLAAIAAIYAIGAETDLSLVTFFLVYMSGFTLAFVYIACLALLLENPLWQRRLRFLQPYGRM
ALTNYLSQTVLCVILFLGFRLYGQVSLWQGTLLCLVIYPLQIAFSAFWLKRFRYGPVEWLWRKWTYGRLSPGRGQSR
>lcl|NZ_CP026520.1_prot_WP_042229469.1_1299 [locus_tag=PC41400_RS06800] [protein=hypothetical protein] [protein_id=WP_042229469.1] [location=complement(1498620..1498946)] [gbkey=CDS]
MSGARGRGTEADCRKRTARQAVSRVRKHGRVRENEPARRQRNRADRIDRTNQIDRPSPGPQRLPMRASGLARIASLPSAS
PRFATGGSFPYDEEKIAFLGYDGTVLLL
>lcl|NZ_CP026520.1_prot_WP_174446810.1_1300 [locus_tag=PC41400_RS06805] [protein=lipid II flippase Amj family protein] [protein_id=WP_174446810.1] [location=complement(1498943..1499731)] [gbkey=CDS]
MLLLACLFTLIIHAAETSAYAFRLAGIRTAKLAVALSLAGMIVLISRTSNMVQGLMISGIADKAGQSDPHLALAPFRFVL
MSASVGTAIAIVLFPTLVSLSARLIAHFEVTGSIPSLLKATANVHGIKQAGKHLRPPRLQMLSRLRIGGVPKRLVLLNVI
VTAIYTVGVLAALYASAYSPDNKSQATMSSALINGIATIIMTLLIDPKVGLLTDRAMHGKASQEAVNKMFGLLMLSRLAG
TILAQALLVPATSFILWALTLF
>lcl|NZ_CP026520.1_prot_WP_042229463.1_1301 [locus_tag=PC41400_RS06815] [protein=iron-siderophore ABC transporter substrate-binding protein] [protein_id=WP_042229463.1] [location=1500111..1501091] [gbkey=CDS]
MKSAARKSYVLCLIVLAMALALAGCGKGNQASDGGAASPAPGDTAKQTIKVKHAMGETDVPAAPKRVVILTNEGTEALLA
LGIKPVGAVKSFTGNPWYDHIKADMEGVTVVGDEHQPNMEAIISLKPDLIIGNKMRQEKVYAQLSAIAPTVFSETLRGEW
KNNFSLYAEAVNKKADGEKVIADFDKRIEDFKSKAGDKLKEKVSVVRFMAGKTRIYLDDTFTGIIFKQIGIARTPVKAKD
TFVEEITKERVPEVDSDRLFYFTYETGDGNGTKQEEAITKDPLWQNLNVVKNGKTQKVNDVTWNTAGGVRAANLLLDDLY
KFYDIK
>lcl|NZ_CP026520.1_prot_WP_042229460.1_1302 [locus_tag=PC41400_RS06820] [protein=response regulator transcription factor] [protein_id=WP_042229460.1] [location=1501284..1501976] [gbkey=CDS]
MYGGTILLVDDETEITNLLEIYLKNEGYRLLKASDGAEALELLNENEVDLIILDVMMPRMDGIEACMKIREEKHMPIIML
SAKSQEIDKITGLSIGADDYVIKPFSPLEMLARVKSQLRRYRQLNSHSAKNENEILIEDLLINLGTHKVYVEGSEVRLTP
REFAILQVLARNRGQVLSMEQIYEAVWDEPFFESNNTVMVHIRKIREKIEKDPSRPQMIKTVWGIGYTIE
>lcl|NZ_CP026520.1_prot_WP_042229943.1_1303 [locus_tag=PC41400_RS06825] [protein=HAMP domain-containing sensor histidine kinase] [protein_id=WP_042229943.1] [location=1502012..1503124] [gbkey=CDS]
MKNKFIHTVRWQFIFIFILSIVFAAGTVLLAYRIVKSIVGLSYYSGPLRWVVNNIGSTPVMVVVGGGLFLVYFFIFSRKI
IGYLEEITQGVQEIAKGNLDYEIVARSTDELGEVAETINVMTRKLKKSIEEERKAERTKNELITGVSHDLRTPLTSILGY
LELIETDRYKDEVELRYYTRIAFEKTKNLNKLIDYLFEYTSLPGEGIELEMKKVNIYGFLKQLAEEFVPILTKAGMTCRV
TAEDDTLYIRADGRQLVRAYENLISNAIRYGSEGKFVDIKISRSGQEAVVEIVNYGEPIPESDLPYIFERFYRAEKSRSS
RTGGTGLGLAIAKSIIEMHGGKIGARSHRKQTVFETRLPLQPSRQPGTDS
>lcl|NZ_CP026520.1_prot_WP_042229458.1_1304 [locus_tag=PC41400_RS06830] [protein=glycosyltransferase family 39 protein] [protein_id=WP_042229458.1] [location=1503283..1504506] [gbkey=CDS]
MNRKKRVWAALAIILLLAAYVRIDFLRSVDHEMPHDSLNYDKMVRQLLETGVYAYNGETPNAFVTPGYPLFLAAVYEIAD
YKEHDPLPWVRYIQALLSLAALGFIYRIARRFSGELPTLIALLIAAVYPPFVWANGAILTEVLAIFLLLGYLLAQIRVFE
TPSYGRAAAAGIWLGLTVIVRPEFLPLIALAFAFQWLRDKEARKRTIRLFACSAAAVVLLLAPWWIRNAVTLHQWVVTAT
QENPFRAGTYPYNTFNDESLVDPVGKTEKELAIARLKAGFTTQPGLFIKWYTIGKMQYVYQSVYRGGGHQPYYPVIPFNA
NLLHRTLMLSGAIAVLAALRRWREPVTLLALVVIAMSLIRLGFVADYRYNVTMMPLIIILDCVLGWRAFMWWKGRRAQKA
AALNPTK
>lcl|NZ_CP026520.1_prot_WP_042229457.1_1305 [locus_tag=PC41400_RS06835] [protein=FAD-dependent oxidoreductase] [protein_id=WP_042229457.1] [location=1504577..1506535] [gbkey=CDS]
MSGSFFTGKKGLLLVLVLVLLAGAGGGWYYYSSKWKAAVGGPLPAKQELTKVVSADKLKDNYDVIVVGTDPEGVSAAVSA
SRNGLKTLLIEPRKRTVLGGLMTEGWLNSIDMNWDKTVSTLPGKEPEYFNQGIFKEWYEKIEGHSFDVTTAANAFNKLVT
DEKNMDVSMDGYSVEPVLKQEGAAQTVTGITVTKQDGTKQTIGTKTVIDATQDADFAAAAGVPFTIGREDIGDKKARMAV
TAVFRLKNVTDEVWSKVAKRLNGDNDKDTGLDKMSAWGYGKEMAGYVPLNKERMKMRGLNIGRQLDGTVLINALHIFNID
GLNAKSQQEGLEIAAKEIPNVVKYLKTFDEFKNVELDSIAPELYVRETRHMEGLYRLSVVDLLENRDQWDRIAFGSYPAD
IQRTSPNDNGAVVVHPTKYAVPFRSIVPQNVDGLLVVGRSASYDTLAHGSARVIPVGMAEGEAAGAAAKLAGEKGTTFRA
MAESKSLIAELQKRLNDQGMDLKPYKLDEAAFMKHKAYPGLKAAVSMGIAYGSYANVGFKELDNPSNAQRMVNQMNSVRR
MYFASFPGDPSAAVKGMEEPAKKTLTLQQASYTIALAAGLKVEAGEAQAALESGGLLSKATLGIIANKEKLSDGDVYMML
KDAVEKQAGAKF
>lcl|NZ_CP026520.1_prot_WP_042229455.1_1306 [locus_tag=PC41400_RS06840] [protein=DUF4352 domain-containing protein] [protein_id=WP_042229455.1] [location=complement(1506741..1507286)] [gbkey=CDS]
MKIKLGLLAAVSCLALTACGDYKEQASPLKQGGGAADRSSAAKAPRFAASLTQEYSKEDTVQIGDAKIKVSKVEKFTEQG
PETPNLPDGEFVVVTLNIENSGAQELAYNPYSFEMANSRGMTSDKVLASFSGETGLKYGELAPGAKVSGTLVFEQPKGDP
KLQLLHKEDTGPDKAVKINLQ
>lcl|NZ_CP026520.1_prot_WP_042229453.1_1307 [gene=trpS] [locus_tag=PC41400_RS06845] [protein=tryptophan--tRNA ligase] [protein_id=WP_042229453.1] [location=1507874..1508863] [gbkey=CDS]
MERILTGDRVTGKLHLGHYVGSLKKRVELQHRMETFLLLADVQALTTHFQDPDLVRRHLRLVALDYLAAGIDPEKAVLCV
QSLIPEIAELTVFYSMFVTVSSLRHNPTTKSEAEQRGYDDLYYGFLGYPVSQAADITFCKATLVPVGDDQIPHIEQTRKI
VRRFNSLYRPVFEEPRALVGDTSRLIGLDGNGKMSKSLGNGIELDASAEEVREKLKKAKTDPSRIRKDDPGHPGICPVFT
YHEAFLAEEAPEIRESCERGSIGCTACKEKLAEGLNALLEPMRERRAHYGAKPAEVDEILQSGTARARGIGRETMREVRE
AMGIDYFGR
>lcl|NZ_CP026520.1_prot_WP_042229450.1_1308 [locus_tag=PC41400_RS06850] [protein=aldo/keto reductase] [protein_id=WP_042229450.1] [location=complement(1509131..1510072)] [gbkey=CDS]
MKYRTLGKTGLKVSVIGVGTWQFGGEWGQDFTQDEVDAVLDKAKELGINLIDTAECYGDHLSESFIGNYLSRRGREDWVV
ATKFGHHYHDRFKRDERFSAEDMIGQLDASLKALGVDYVDLYQFHSGTDSAFDNDALWTALDKQKQAGKILHIGTSIAKN
DNLHQVAASTQVGSEAIQVVYNRLDRAPEESVFQSCLQQNLGVLARVPLASGYLSGKYKPGATFAATDTRHRHDPAETQR
KLEEVQRIREHEVPEAMDMATWALAWCLKHDAVSAVIPGCKNVAQVESNAKAAQYLSDNHPQAWTEEAEASGA
>lcl|NZ_CP026520.1_prot_WP_042229449.1_1309 [locus_tag=PC41400_RS06855] [protein=TetR/AcrR family transcriptional regulator] [protein_id=WP_042229449.1] [location=1510241..1510810] [gbkey=CDS]
MNTPDRIKAAALDLFASQGYEGTSMANIAEVVGIKTPSIYAHFKGKEELFLSVLQDASRGELDFAASHLETGGEALDRKL
YKLLSHTRGKYEHNGSTKFWIRMMYFPPASVYRETIDLSYTYLSRMEELLTSVFTGDRSAIGVDPDKAAAAFLCLMDGVI
VELLYGTAESYDKRLEASWDVYWRGISHS
>lcl|NZ_CP026520.1_prot_WP_042229448.1_1310 [locus_tag=PC41400_RS06860] [protein=multidrug efflux SMR transporter] [protein_id=WP_042229448.1] [location=1510854..1511201] [gbkey=CDS]
MNRNWIYVLAAGLLEIGWVSGLKHAHDAFTWILTGAAILLSFWVLLMATSRLPVGTVYAVFTGIGTAGTVLTETIVFGEP
ISWLKIVFILILLGGVIGLKITTKEPEEPAKGGNS
>lcl|NZ_CP026520.1_prot_WP_042229446.1_1311 [locus_tag=PC41400_RS06865] [protein=multidrug efflux SMR transporter] [protein_id=WP_042229446.1] [location=1511201..1511515] [gbkey=CDS]
MAWAALILAGCSEIFGVLAMNKIARKKTIGAYVLLFFGFGISFSLLSYAMETIPMGTAYAVWTGIGTCGGTLLGMFMFGE
SKDWRRILFIAMIISAAVGLKLLT
>lcl|NZ_CP026520.1_prot_WP_042229445.1_1312 [gene=pheS] [locus_tag=PC41400_RS06870] [protein=phenylalanine--tRNA ligase subunit alpha] [protein_id=WP_042229445.1] [location=1511963..1512997] [gbkey=CDS]
MRERLEALKVQALQELAAVGSQQELNDLRVKYLGKKGPLTEILRGMGALSAEERPVIGQVANDVRGAIEEVIAAKQEEYD
RQETENRLRAETIDVSLPGRPSVQGAVHPLNKIVEEIEDIFIGMGFTVVEGPEVERDYYNFEALNLPKDHPARDMQDSFY
ITEELLLRTHTSPVQVRTMEGRKGAVPVKIISPGKVYRRDDDDATHSHQFTQIEGLVIDKNIRMSDLKGTLLQFVRELYG
PQTQIRLRPSFFPFTEPSAEVDVSCMMCGGEGCRTCKQTGWIEILGSGMVHPKVLEMSGYDPNEYSGFAFGMGPERVAML
KYGVEDIRNFYTNDLRFLKQFVHL
>lcl|NZ_CP026520.1_prot_WP_042229444.1_1313 [gene=pheT] [locus_tag=PC41400_RS06875] [protein=phenylalanine--tRNA ligase subunit beta] [protein_id=WP_042229444.1] [location=1513023..1515473] [gbkey=CDS]
MKVSYQWLSDYVDVQGFTAEDLADKMTRSGIEVDIVESLNKGVSGVVVGYVKSKTKHPDADKLNVCTVDVGTGEDLQIVC
GAKNVDAGQKVPVAVIGAVLPGDFKIKRAKLRGVESQGMICSAKELGMNDKLLPKEIQEGILVLPENTEIGKPIVDVLGL
SDNVLELELTPNRSDCLSMIGTAYEIGAILGRPVKLPEEGRNINVDFSTVKAEDKISVSIAAPDQCAHYAARLIEGVRVG
TSPLWLQNRLMAAGIRPINNIVDVTNYVMLEYGQPLHAFDADQVAGGKIEVRLATEGEKLVTLDDVERTLEPHMLLITDA
FKPIAIAGVMGGANSEVTEGTTSILLESARFAGPSVRKTSRQLGLRSEASLRFEKEVNPEAVIPALDRAAALLVLCAGGQ
VAQGIVQETVQAYSGVKIPLKLERVNTYLGTELSLQQVKEIFDRLGLEYGLLEEDSLIVHVPGRRGDITRDVDLIEEVAR
LYGYDNIPTTLMTGVTTPGSLTREQSLRRTIRRLLTGSGLHEVITYTLGQRENAADYPGLYPEYAPIPLAMPMSEDRSAL
RTSLVPHLLEVVSYNRNRNNDDVAIFETGTVFLSKETALTAQPQEKRLLSIAMTGNWSGAHWGQKARKADFYDLKGVLER
VISYLGVTIEFRSAQKEGFHPGRTAELFLLTPDGERRSIGYMGQLHPTLQKKRDLEDTYVLEAELDPLLQHADFEVDYRI
LPRYPSVGRDMALVVDTAVPVGDLERLAAETASELLESIRVFDVYTGERLGEGKKSVALALVYRHPERTLQDEEVTETHA
RVVAALESAFGAELRK
>lcl|NZ_CP026520.1_prot_WP_042229441.1_1314 [locus_tag=PC41400_RS06880] [protein=hypothetical protein] [protein_id=WP_042229441.1] [location=1515653..1520623] [gbkey=CDS]
MNSEEKTKIIVDIYGNEYKLKANNSPGYVRKVAAEVNDQMVQIAESYPRLDTYKIAVLAAVNLTDEKAKVEQRFSETESR
LNDLQRALSKADAAVETARGQLERAQQEEQLRKAELAEARELGAKAAKRQQELTASLEQTKQRAEQNLRELNAAKEKAQQ
AAKREEQLKSELQQARQNTEKLQREQASLKQQMEKDARQADQWKKELEQAAGQIKTLESKIGQAERDAESLLEQLEQEKT
AAGQAREESAAALLQIQEAAEKEQTLRAELQLVKQRSEQVQRDLQASKQREQQQNGQMEQLRGELAEAAQREAAVRKEAD
QAKQDAEALLEELELEKAAAEENRQQAASELEKLAREQKHTDEQLRQLKDELELAKLEADESREREERLNADAQQAAERE
EMLRSELQQAQRDAEALLEEAEREKAAADSLKEQLREAGEREQTQRTELALMKQRGEHMQRELAQGKELAASREQKLRET
ADQLEEIREELEKSRKDAEALLEELEANKASAEEELRESTEALKAARLESKAVREAAETEKAAAEMRLRNAEEALVTARR
EAEESLAADRLEAAEKLEVLQKEAAERLELAQLEAEEQLEALRREAAHLEAARKAADAQLEALKRQSAEELELAQMEAEE
KLEAARREAAEKLAVAENESADKLLTAVREAEEIEAARKEAAAKLQEAQREAAVAREAFRQEMEAARERAEQELLEAMEA
AEQEGEARKEAQRGREAAEAKLREASEALQALELKASGLEVEKQEAAARLEALERETADRFEALKRQGAEELELAQLEAE
EQLDAARREAAAELEEARREAAAQLQAAASEAQELEAARKEAADKLAEAQREAEAAREAFRQELAAEREKAEQELLEAME
AAERETEARAEVQREKDDAEAKLREASEALQAAERKAAELEALMETAAAQLEVLKRESADELELAQLEAEERLEALRKEA
AAEREAAESAAAEKLQAAERKAEELESARKETADMLEAARREAEAAQEALRRELAEEREKAEQELLEAMEAAERESGARL
ELQREKEAAEARLRETEQALTVVRQEAGELAASRSEAAEELAAARRDFAEQLEFAELEAAEKLASAQREAEAAGRRELAA
ERELAAERERAEREQLETAGAAERDRAAREAAEMEKAALEATLRETSEALEAARQEAVQARESAAKLVETLRAELMGARQ
DAEALLEDLEQDKTAADEELRKVSEALRAAQEEAAAAASSANAAKAAERDAEARTVSVREEMAVRLRDSEEREEALRAEL
EQARRDAEALLEDLEQGQTTADERLREAEKALEEARLEVERAREDRAKLEASLKEELTETLKQGELMLTQVAAAREKADK
ELASERGRSAQERLAAAELAAQLEGALKREETLRTELNQARQDAEALLEDLEKGQGAAAAKLQETAAVLEAAREAAVREL
AAERDKSAGERRKAEELAEQLHAAVSGRDEAAAREAKLREELNQAKASGEALNAELTAALREVAATQQMAHEQVLQEQSI
RRELESSMQERRSLQEELDAYRQLMDEQADQERASLGELGGLRGETEQLRQEVARITEELETEAKRSGLLRHQLEELEQT
HLQAEQSLKEMSRLHEQTSGRWDELQEEHRKLQDEYAKLQTEYNEWIQLVMEEPKG
>lcl|NZ_CP026520.1_prot_WP_042229440.1_1315 [locus_tag=PC41400_RS06885] [protein=CvpA family protein] [protein_id=WP_042229440.1] [location=1520623..1521171] [gbkey=CDS]
MTLPPWNVLDWSILVLIAGGLVLGYARGLISQLVSIAGLFIALYVAVRFYEPVATYLLQWLSLSSFSGYDKYEFLVKGLN
LERYIANALSFALLFFLVKLALGIGGRMLNLIARLPGLHTVNRVTGALLGAIEAFVIAAVAVHIMTILPSDYVQDRLGES
RFAPYILNDVMEWARKLWQRSM
>lcl|NZ_CP026520.1_prot_WP_042229439.1_1316 [locus_tag=PC41400_RS06890] [protein=hypothetical protein] [protein_id=WP_042229439.1] [location=1521387..1521962] [gbkey=CDS]
MEQTLAAGSVVTTACILAAACLGWFYVWIRRKQKAEGAALLQSVITALAVTFFFLLPLLIMFVCVYAALLSAGTFLPDSI
WFQQSEDLLSLALLLIFAQTLLSIPVRLLLRLGALKGLGEFRKKIGMPLVPLQLSLTMLTFALGYMAAREWHLTSVDLTE
SGLYYVSILPPLAAFGIFTLLRRRRENPPGA
>lcl|NZ_CP026520.1_prot_WP_009676126.1_1317 [locus_tag=PC41400_RS06895] [protein=cupredoxin domain-containing protein] [protein_id=WP_009676126.1] [location=complement(1522167..1522535)] [gbkey=CDS]
MKKTAALLMALVLVFALAACGKKTDTAATDANATASQEITLQASNWKFDQAEYKVKKGEPLKINLDIKEGVHGVSIPDLK
VSLDNNSKSKVVTPEKAGTYNILCSIPCGSGHSTMTAKLVVE
>lcl|NZ_CP026520.1_prot_WP_042229438.1_1318 [locus_tag=PC41400_RS06900] [protein=phage holin family protein] [protein_id=WP_042229438.1] [location=complement(1522704..1523069)] [gbkey=CDS]
MRFLGHVIRFIVSALVLMVVSWIVPGFSVGGFVSALLLAIVIAVLAWIIESIFGRQITPFGRGIVGFLTSALVIWLSQFI
VTGVHASIIGAILAALVVGIVDLFIPVSPFDRKDSRRKSTH
>lcl|NZ_CP026520.1_prot_WP_174446808.1_1319 [locus_tag=PC41400_RS06905] [protein=endonuclease MutS2] [protein_id=WP_174446808.1] [location=1523336..1525711] [gbkey=CDS]
MRENALNTKTFNTMEFHKITHKLSSHAATSLGTAKAAEVKPSGDFEEVKLRLQATDEAFAVDRLKGGAPFGGIRDIRAAV
HRARVGGMLNPSELLDIATTIYGGRRLRKFLETMDEQHPVPRLLSWAELLSDNKETEDRIKSCIDDNAAVMDSASSELSR
VRSELRTGEARVREKLEQMIRNPSVQKMLQDALVTMRGDRYVIPVKQEYRSSFGGMIHDQSASGATLFIEPEAVVTLNNR
LRELKFKEEAEVEKILRMLTAGVAEIEEDLLSDVELLAQLDFTFAKAGLAREMKATLPKLNDRGFLKLKKARHPLIPLDK
VVPIDLELGGSFTTIIVTGPNTGGKTVSLKTVGLLSLMAMSGLFVPAEEGSELCVFDGIFADIGDEQSIEQSLSTFSSHM
TNIIGILRDMTPKSLVLLDELGAGTDPAEGSALAISLLENIHRMGCRMIATTHYSELKAYAFERRGVINASMEFDVQTLS
PTYRLLVGVPGRSNAFAIAERLGLPRAIIEKARGQVSDEDQRVETMIASLEENRLTAEAERHSAEQIRRENEELRAALSA
ERAKLDEQRDKVLLKAEREAQDAIAKARREAEEIIADLRRLAKEEAGAIKDHKLIAARRRLDEAEPKLRKPGAKRPERSK
AEVEPGDEVRVVSLGQKGHVVEIVNSSEVTVQLGIMKMKVNKTDLEVIKASPQKKPVQQVAATVKRTRDDNIRMEVDLRG
TNVEEGIMELDRFLDESFLNNFAQVYVIHGKGTGVLRTGIQDYLRRHKHVKSFRIGAYNEGGHGVTVVELK
>lcl|NZ_CP026520.1_prot_WP_042229435.1_1320 [locus_tag=PC41400_RS06910] [protein=PspA/IM30 family protein] [protein_id=WP_042229435.1] [location=1525834..1526508] [gbkey=CDS]
MGIFKRLRDLTMASMNDLLDKAEDPVKMLNQFLRDMEQDIQEAEVAVAKQIAIEKKFKQQYEEAEEMVGKRTEQAMKALE
QGNEDLARRALEDKKEHQTRFDEMKRQYDIAKTNADQLRNQLSDMKDEFSKMKNKKDLLIARAEAAKAQKHINQAMSGFG
TDNAAKGFDRMSEKVLQLEAEAQASGELRSKGKSLDDELDQLGKSDGIDDELAAMKAKLAEKKQ
>lcl|NZ_CP026520.1_prot_WP_009676216.1_1321 [locus_tag=PC41400_RS06915] [protein=DUF350 domain-containing protein] [protein_id=WP_009676216.1] [location=1526604..1527023] [gbkey=CDS]
MYEEVNQLVSNPYLGALAYFSVAVVAVIVFLTIFEIVTKYNDWEEIKRGNVSVAMATGGKIFGICNLFRFAIENNDSLIR
SLIWAGYGFLLLMAAYFIFEFLTPYFKIDEEIQKDNRAVGLLSLMISVSISYVVGACVP
>lcl|NZ_CP026520.1_prot_WP_042229429.1_1322 [locus_tag=PC41400_RS06920] [protein=hypothetical protein] [protein_id=WP_042229429.1] [location=1527181..1527705] [gbkey=CDS]
MNEEELKTVCPWCQTEIVWDPEIGPEEECPHCYNELSDYRSIRLGADEEDADEPDEEAVPAAGSDDFLDWEEEDSYTNEY
AEAVRNVLDTQEEAPECSACRELMLHGGTQVTAGTGYAPVIPGTVGKAFLNPGYKLNVFVCPSCFKVETFLAEEDREKLL
AVLNPSKEQPATDN
>lcl|NZ_CP026520.1_prot_WP_042229424.1_1323 [locus_tag=PC41400_RS06925] [protein=MFS transporter] [protein_id=WP_042229424.1] [location=1527979..1529310] [gbkey=CDS]
MQQDKERGRSRLAGWLGTAKETPAAESWEHEADAAASAGEKKKPTIGRLDAQSALLLIENMLFAAANALSGTFVSIYLWK
AKSDFAMIGWFTLTHQLAMALTFWVAGKWVKEHNKMNCLRAGVVMSAVFYLLVLMLGLSSVNYIFLLGAVQGMAAGFFWL
AFNIVYFEVTSRETRDKFNGWAGLLGAGVGMLAPWISGLIIVHMAGTNGYRVIFSISLGIFVLGVAVSFFLKKRKIEGTY
GWTYGFRHLRQGDTPWRPVCAALMAQGFREGVFGFMISLLVYIATTSEQKLGSYALVTSGVSFVSFWLVGRVLKPKYRSR
GMLLGVLMVIAAILPFFWKVNYMTLLAFGIGTSLFLPLFTIPMTSSVFDLIGRDEESVRHREEFVILRELALNAGRILGT
LLFIAVVSQSTSAMVINTLLLVIGSAPFLSWLFMRKVLHPENS
>lcl|NZ_CP026520.1_prot_WP_042229423.1_1324 [gene=cysK] [locus_tag=PC41400_RS06930] [protein=cysteine synthase A] [protein_id=WP_042229423.1] [location=1529405..1530331] [gbkey=CDS]
MSGKLVNSIAELIGDTPAVRLNRLTSEEHAEVYVKLEYFNPSGSVKDRAAYNLIAQAEKDGLLKPGATIIEPTSGNTGIG
LAMNAAVKGYKAILVMPDNMTKERINLLKAYGADVVLTPAAERMPGAIAKALELQRGIPGSFIPQQFENKANPDIHRVTT
ALEILEQTDGRLDAFVASSGTGGTITGTGEVLRERIPGIEIYVVEPAGSPVLSGGEPGPHKLVGTSPGFVPKILNTDVYD
EIVRVEDEDALRTTRELAAKEGILVGPSAGATVWTAIQVARKLGPGKRVLCIAPDTGERYLSMGIFGE
>lcl|NZ_CP026520.1_prot_WP_042229420.1_1325 [locus_tag=PC41400_RS06935] [protein=GNAT family N-acetyltransferase] [protein_id=WP_042229420.1] [location=1530920..1531228] [gbkey=CDS]
MISLSGEMPKPWVLNPFIQSYTELTSQKVDRKSLKQIGNGRFISAYEDQKLVGIAFCTGDVEWENEVSLSVLILPSYRKR
GIESAMLRLLRGKQSSYSRTVG
>lcl|NZ_CP026520.1_prot_WP_042229418.1_1326 [locus_tag=PC41400_RS06940] [protein=Lrp/AsnC family transcriptional regulator] [protein_id=WP_042229418.1] [location=1531601..1532092] [gbkey=CDS]
MDDLKHKILDILKEDSRRSPALIATMLGITEDEATAAIRQMEDEHIIVKYATVLNWSKVDSDKVNALIEVQITPERGTGF
DAIAERIYLYPEVKSVYLMSGAYDLLVEIEGRTLREVSSFVSSKLSTLDRVLSTKTHFILKKYKQDGIIFDDREDDHRML
VSP
>lcl|NZ_CP026520.1_prot_WP_042229941.1_1327 [locus_tag=PC41400_RS06945] [protein=aminotransferase class I/II-fold pyridoxal phosphate-dependent enzyme] [protein_id=WP_042229941.1] [location=1532105..1533301] [gbkey=CDS]
MINQEEIGSKPKSMQGYLSPLVRDIPPSGIRRFFDLVSGSKDIISLGVGEPDFVTPWHVREACVYSLERGRTNYTPNAGL
PELREAIAEYLYTNFAVKYEPSNEVVVTVGGSEAIDLALRALIAPGDEILIPEPCYIAYSPIVTIGGGVPVGIETFAKDS
FKLKPEALRASLTPKSKVLVLSYPSNPTGGIMTYEDWEPIARIVEENDLIVIADEIYAELTYGSKHVSFASLPGMRDRTI
LVSGFAKAFAMTGWRVGYMCGHPDLISAMLKIHQYTVMCAPIMGQIAALEALRNGLEEKDRMIESYNQRRRLVVQGFRQI
GLDCHEPQGAFYAFPSIASTGLSSEEFAQRLLTEAKVAAVPGQVFGNGGEGFVRCSYATSVSQLNEALDRIGSFVKKL
>lcl|NZ_CP026520.1_prot_WP_009676194.1_1328 [locus_tag=PC41400_RS06950] [protein=aspartyl-phosphate phosphatase Spo0E family protein] [protein_id=WP_009676194.1] [location=1533372..1533632] [gbkey=CDS]
MAYLEYRLADNSGWYSHDVSDQRWLSKRPAKLASLFHLEEEIQFMRHRLEQMVQSGEAMTSESVIEMSTLLDHKINEYMN
LVQKSR
>lcl|NZ_CP026520.1_prot_WP_042229940.1_1329 [locus_tag=PC41400_RS06955] [protein=hypothetical protein] [protein_id=WP_042229940.1] [location=1533687..1534262] [gbkey=CDS]
MTQKRWFPAAVLLLLFTVLVSGCGKAQPKADVSIFMMARDGIPVDAAFKLENSLKALVGDKPSIYLNTSPLFSNEKMIVE
MGYGNHGILIIPEEQFKIYAAEGAFLPLEEHFKQEDYPEGVLEATIEPAGKKKNPPEAPKKETHLYGIPLQYSKWMKDNG
VEGKGLVAFMAVNSRNPEQSFEVLKKIAAKS
>lcl|NZ_CP026520.1_prot_WP_042229417.1_1330 [locus_tag=PC41400_RS06960] [protein=ABC transporter substrate-binding protein] [protein_id=WP_042229417.1] [location=1534773..1535780] [gbkey=CDS]
MNNERKTTGGFKYALLLLVLTMVVGLAACGKQSADNGTAGSASPAASTAPQSSAKKTQYPLKVTDATGKEFTFEKAPGKI
VSVSPAETEALFAIGLGDIIVGVSDFDDYPAEAKTKPKMGGITQPNQESVIASGADIVFTGVSMKKESVEKLRAMGINVF
KVEPKTLDDVMKNIQLYGQITDHQEQAEKVVAKMKEDRQKVTDAVKTVAADKKKKVYIEFSPGWTVGKGEFMDELITLAG
GVNVASDITGWKQISEEKIIQQNPDVILFAKGVVDDKTKKSLEELIRGRSGWNEINAVKNNQVIGLDQNTLTRPGPRLTE
GLLQMAKAIYPDLVK
>lcl|NZ_CP026520.1_prot_WP_009676079.1_1331 [locus_tag=PC41400_RS06965] [protein=iron chelate uptake ABC transporter family permease subunit] [protein_id=WP_009676079.1] [location=1535781..1536818] [gbkey=CDS]
MKPKWGLWGGVGVLLLLLSIVLSLSLGSAHLPVSQIASILGSKLPWIGDRITPDWPVSSEQIIMKVRLPRVMLAILVGAS
LSVAGAAFQGVLRNPLADPYTLGVSSGTSVGAAFVIYFGLQTTFFGTWTIPVVGFATGAVSLWIVLRLARTDGKIRMETL
LLSGVVMQAFLGSVVSFLVSMSKQVVNEIIFWLMGSLAMKGWSYTTVLLPYFLVTFVVLYSYARTLNLLALGERQAAHLG
IHVERTKLIVLITATLVTAAAVSVSGVIGFVGLIVPHLIRLIAGPDYRLIVPLSAVGGAIYVLWADTLARTLLSPTEIPL
GVITAFLGAPFFAYLLRKQKKWGRS
>lcl|NZ_CP026520.1_prot_WP_042229416.1_1332 [locus_tag=PC41400_RS06970] [protein=ABC transporter ATP-binding protein] [protein_id=WP_042229416.1] [location=1536818..1537603] [gbkey=CDS]
MEPIIEVQGAGKSFGGIRVLKDIRFEVKPGEFFGVIGPNGSGKSTLLRLISGVDPLTEGRILLQGREVAGYARKELARWL
AVLQQDALPPLGFTVREVVEMGRFPFQNWLGEDREDAAALIDRIMRRLDLEPLADRTVENLSGGQKQRVALAKVMAQQPR
LLLLDEPTTYLDIGYQMQLMDYVRQWQREEGLTVVAVLHDLNLAALYCERLLVVHEGEGISIGTPEEIMRSELILDVYGT
EPIVLEHPDHGIPQILLRRGD
>lcl|NZ_CP026520.1_prot_WP_042229415.1_1333 [gene=cobT] [locus_tag=PC41400_RS06975] [protein=nicotinate-nucleotide--dimethylbenzimidazole phosphoribosyltransferase] [protein_id=WP_042229415.1] [location=1537679..1538728] [gbkey=CDS]
MNELQDVIGAIRPLRQEAKDAASVHLDNLTKPPGSLGKLEDMARQLAGITGEVMPDMSRKAIVVMAADHGVCEEGVSAFP
QEVTRQMVMNFLQGGAAVNVLARHAGAEVVCVDIGVNADLEHPDLLSRKVRKGTANMAREAAMTREEALQAIRVGIEVAD
GLAGRGVRVLATGEMGIGNTTASSALLSVLGGIAVEEAVGRGTGIDDARWLHKQEVVRRAIEVNKPDADDALDVLAKVGG
LEIAGLTGVILGAAKNACPVVIDGFISSAAALVASRLAPESQPYMIASHQSHEQGHGRMLEAVGLGPMLHMDMRLGEGTG
AALCFTLMDAAGKIMREMASFESAGVSRG
>lcl|NZ_CP026520.1_prot_WP_042229413.1_1334 [gene=cobU] [locus_tag=PC41400_RS06980] [protein=bifunctional adenosylcobinamide kinase/adenosylcobinamide-phosphate guanylyltransferase] [protein_id=WP_042229413.1] [location=1538728..1539321] [gbkey=CDS]
MNLAQEHKPGAGAVLVTGGARSGKSSFAEQLSSRLAAHGTYLATAQIWDEEMRQRITLHRTQREQSGFPWITAEEPMELS
AALKACSGERGRGVVLVDCLTLWLSNRLLKEESSPDAAEQVNAAVDELVRTVEAYEGPLVLVTNEVGSGIVPEYPLGRLY
RDLAGRMNQRLAQVCGRVFLVTAGIPIELKSRAYRFE
>lcl|NZ_CP026520.1_prot_WP_042229410.1_1335 [gene=cbiB] [locus_tag=PC41400_RS06985] [protein=adenosylcobinamide-phosphate synthase CbiB] [protein_id=WP_042229410.1] [location=1539331..1540458] [gbkey=CDS]
MIFYSLKEILWMMLAAVAVDLVIGDPKWPTHPVIRIGSLISFLERKLRKEPASRKSAPECTGGGQAEAGAETGAETRAVA
GAAEGVPLEAGAPLDTAGQAAALSPRSLKRRGIVLAAAVMLAAFAAMTAVTAIVAAIHPWLGYAVTTWFISTTIAVKGLK
DAARLVAVPLAAGDLAAARKYTGYIVGRDTKNLDEGELTRAVVETVAENTVDGVVSPLVFALIGGAPLAMLYRAANTLDS
MVGYKNDRYVHFGWASARWDDVMNWIPARLTGILLILAAAVQPRMSAFRAARSVGRFARLHPSPNSGIPESAVAGALGIE
LGGRNTYGTVVSERARMGWPLRPRNRGDIDGTVRLLYGATGWLAGGVLCAIWLLN
>lcl|NZ_CP026520.1_prot_WP_042229409.1_1336 [gene=cobS] [locus_tag=PC41400_RS06990] [protein=adenosylcobinamide-GDP ribazoletransferase] [protein_id=WP_042229409.1] [location=1540434..1541291] [gbkey=CDS]
MRNLAAELKLWLFGLVTAIQFLTRLPLPVRIEWTPAVLRRSLAFYPVVGMILGLLLTGSGWVLEKALPPLPAAALLLCLW
IGLTGALHLDGLMDTADGLLSHRSRERMLEIMKDSRVGAMGVIVCVAVMVLKWSVMVPLLTGGALTGSWILLLVPVWSRT
FMAAAVALWPYARSGGGLGGFFEGEKKPFALLGAGIALFLSAGAVAFFRQSGGTLPLKPSLHVLEGAGPVLWGIMLLMVT
FALGGVLAVWMAKKLGGLTGDTYGALNELLEVLLLLAAAFVLHLT
>lcl|NZ_CP026520.1_prot_WP_042229408.1_1337 [gene=cobD] [locus_tag=PC41400_RS06995] [protein=threonine-phosphate decarboxylase CobD] [protein_id=WP_042229408.1] [location=1541469..1542584] [gbkey=CDS]
MLERYGHGGDVWTAEEKFGRGREQFLDFSSNMNPFGPPDAVEEVLRSGWKNIVRYPDPAVRELRGKIAAKYGVPAESVLV
GNGAAELIDLIVRVLQPEVTALARPSFSEYEDAVLKTGGRIHGIPLRAEHCFRLQKADAEEALKRSDLLFLGQPNNPTGR
LIPQELLNDLAGSGHPLILDEAFLDFVPDEKNVTLLRLAAESRHVAVIRSMTKFYAIPGARLGFIVAHPDVIGRMSRLQV
PWSVNYFAQLIGCAVLEDEAFHAKIISWLQEESPWLAAGLAGLGLEVTPSDTNYLLCALPEQSPINVRILQEEMGRRGVL
IRDASLFEGLGTRYFRTAVKLRSDNERMLAALSDVLKAAQAGEISLERGGR
>lcl|NZ_CP026520.1_prot_WP_156972816.1_1338 [locus_tag=PC41400_RS07000] [protein=cobyric acid synthase] [protein_id=WP_156972816.1] [location=1542587..1544269] [gbkey=CDS]
MRNSTHTEGRKGTVSTHRDREADQPEASGSGTDGTRARTLMVQGTASDVGKSILTAALCRIFVQDGHRVAPFKSQNMSLN
SYVTPDGKEIGRAQGVQADACRIPATTDMNPILLKPSQDMRSQVVVHGKPYQDMDARAYRESYLPLAEKVVCEALERLRR
TVDLVVLEGAGSPAEINLKDRDIVNMRMAKWADAPVILVADIDRGGVFASIVGTLELLEPDERERVKGFVINKFRGDVSL
LKPGLDWLEERTGKPVLGVVPYYPKLGIEDEDSASLDSKRRLEPAPAAQQDGQRPDVLDIAVVRLPRISNFTDIDPLHEE
PDVSVRYVGSAAEWGNPDIVLVPGSKNTMSDLLFLRESGLADKIVRHAGSEGRVAGICGGYQMLGEALSDPLGTESDYRG
DREGLGLLPVATEFRPEKKTVRAEGSCGLFREKPDLPVDGYEIHMGVTRMTDETSERPFRLRASLTGSGDEAGAAEGLHP
EGASAFEGRVWGTYLHGILHNDDFRRCWLNEARQSKGWEPLSAELRFREKREEAFDRLADHVRNHIDMDLLYEMAGLTAE
>lcl|NZ_CP026520.1_prot_WP_042229405.1_1339 [locus_tag=PC41400_RS07005] [protein=cob(I)yrinic acid a,c-diamide adenosyltransferase] [protein_id=WP_042229405.1] [location=1544447..1545016] [gbkey=CDS]
MKIYTRTGDAGKTGVIGGRVDKDDERVEAYGTVDELNCYLGQAMSFMDPEKFGDLLEDLLTVQHELFDCGSDLALLKQEL
RPFKVTAEMVDNLETWIDKYDAETPDITRFILPGGSRGSAALHICRTVCRRAERRVVTLARTQEINVEVRRYLNRLSDLF
FTIARTANFREGQTDVEYVRSAEVFRRKK
>lcl|NZ_CP026520.1_prot_WP_042229404.1_1340 [locus_tag=PC41400_RS07010] [protein=RluA family pseudouridine synthase] [protein_id=WP_042229404.1] [location=1545013..1546155] [gbkey=CDS]
MMNYYSPLTYVVPPDEEGFYLKTILQNRMGLSRKLLSRLKLTEQGITVNGKREYISIKVNGGDVIEIRMQEEESEDILPQ
DLPLEILYEDEHLLIVNKESGMIVHPTHGHYLNTVANAAVHHWKERGENVRFRPVHRLDQETSGVLAIAKNPYVHQQISE
QMIAHSITKEYLAIVHGTVAEDRGTIDAPIDRDPEQPHVRIVTPSGYRAVTHYETVRRYDGITLVRLMLETGRTHQIRVH
MKHLGHPLLGDKMYGPEADAPARKSEGSPSGEARDAAEGCAAGEGVYGAPEADSAAGEPARPPAADFAAGEAPGADVSGG
MPGAPSAAPSPAADDAAGPALARQALHAYRLGFVHPGTRERVEFTAPLPADMQAVLDACE
>lcl|NZ_CP026520.1_prot_WP_042229403.1_1341 [locus_tag=PC41400_RS07015] [protein=arsenate reductase family protein] [protein_id=WP_042229403.1] [location=1546512..1546868] [gbkey=CDS]
MSVTIYEYSKCSTCRSALKWLRAKGLDPVTVPIMEQPPTVEEIRVLVRNSGLELKKFFNTGGEVYRELGLKDKIGSMTED
EQLALLASNGKLLKRPIVTDGSKVTVGFKEADFEKAWG
>lcl|NZ_CP026520.1_prot_WP_129112457.1_1342 [locus_tag=PC41400_RS07020] [protein=site-2 protease family protein] [protein_id=WP_129112457.1] [location=1546908..1547990] [gbkey=CDS]
MTELPDKQSSKSSKKNPLWVLGAIGAFLLKFGKFGKFGATIISIAITIASYAIIAPLPFAIGLVAMIFIHEIGHVLAAKK
KGLPVTAPLFIPFLGALISMKRNPRDAVTEAYIAYGGPLLGTIGATAAFALGVYTDSNLLISIAYTGFFLNLLNLLPIHP
LDGGRISTAVTRWLWLVGLIGGLVVILYLRSFLFLIIWALFAWDLYKKFVKFRDKPQIMHFQPRFDMTAHHLLAHGYMIP
GEDHRRDLAFTTYSELSGDEAGVQMVHTSWDAIEFETKFALPRQAVVHRVQVVKIERVTREDGLHLVLHFDVELEPFEND
AYYEVPTASRWKFGAAYFALAVYLLAMMNWVHRVLPAAAY
>lcl|NZ_CP026520.1_prot_WP_042229401.1_1343 [locus_tag=PC41400_RS07025] [protein=aminotransferase class I/II-fold pyridoxal phosphate-dependent enzyme] [protein_id=WP_042229401.1] [location=1548020..1549213] [gbkey=CDS]
MKAYGSTRLDSLGSAIFAEVAQWRREAEERGVDVIDLGIGSPDLPPSPRVIEALADAVRNPRAYGYPGSEGTPEFRRAAA
GWMHRRFGVTLDPQREVLALMGSQDGLAHLAMALCDPGDTALVPDPGYPIYAASLVLAGVQPVLMPLRESNGFLPHLESI
PPETAEKASFLLLSYPGNPIPEAADRAFYEKALAFAERYDLLIVHDLAYSELAYDGYKPMSILELEGAKKRAVEFHSLSK
SFNMAGCRIAFLAGNAEAVGALRRLKSNIDYGVFLPVQQAAVVALEEETSFPSSISEVYERRRDALVDGLRQIGWHVPRP
KATMFVWAKIPQGWTSRHISREILERAGVAMIPGDAFGAEGEGYVRIALVQSEERLREAVSRIGEIWQSISERERNT
>lcl|NZ_CP026520.1_prot_WP_042229398.1_1344 [locus_tag=PC41400_RS07030] [protein=DNA sulfur modification protein DndB] [protein_id=WP_042229398.1] [location=1549310..1550446] [gbkey=CDS]
MDSIFSLSELLSQQKPSSLVEIPGVLGQTFGQATLSTTLPISKLFSIYEVDLEVQRAIIPRNLSKLMDYVSLYLDNHQPI
YFPGVIFSARGAGSYDSEQHVFQLQPIEKLYVVDGQHRLAAFQRLMETLQSQMARAKDRREYDRMEVITEKLSRLYAFPV
STMIYLDIDARQERQLFSDINKLPRKIGGNLAVLRDQRRFYHVTASQLVEKAPVMQHIPVDMFSERGKAPEFLFTYHLLI
EILVALFEGRMKSGARNNGYHYEEEEVTDHLSISSFYFQQLLKYLPEPVKGEPAWSENIQIALALFFHEEATKSAKFNRY
ALEYATKILPHIDWNAIFSGDEKDRLPRRSRIMKAYQYIRSFYQEQHLFLISDKEDVG
>lcl|NZ_CP026520.1_prot_WP_042229395.1_1345 [locus_tag=PC41400_RS07035] [protein=DNA sulfur modification protein DndB] [protein_id=WP_042229395.1] [location=1550446..1551618] [gbkey=CDS]
MDGLCLRMTIHPFCERFGLATVALKVQDLLNYTMIDAIVQRKLSKMQRRKISNYLQERELDQVFFGPVTLSLRDVSQLTR
TSEELYLRHGSKLSIIDGQHRILALGFVNEQMQKEVKRTERKLASLRVRHRKETDNRELADEIAQVSGLLEQMEARRLGL
MESELAVQIYIGLNEVEEQQLFGDINSKVQLVSKELGHSFDSVDPLNLVIQQVVDHNMLLKEAGVEKRSNLTAFNRNFTC
LSWMYSTAVMLFSGKMQSSYELQRKIRKETATYVEILHQFYNTILPMMPEQPGLVQVTSANRVMQESVALYANQFLFRDG
VYNPEWTSCLRILEGFDWSHGNEELVYIFGSLDNGKLNLIHEKSLRKHGKLVSFFLEGIEPDGLKSAELA
>lcl|NZ_CP026520.1_prot_WP_042229393.1_1346 [locus_tag=PC41400_RS07040] [protein=glycosyltransferase family 39 protein] [protein_id=WP_042229393.1] [location=1551846..1555118] [gbkey=CDS]
MQIGETMPKEANVRHAIFALALAAAVLIRFLVAPNVFGYDADVRTFMAWADRAFTVGFGNLYTDPGYFLDYPPGYMYILY
LIGAAHHLFGIGWGTGLSLVLLKTPPMLADLLASVLLYRLALRSANVRQAAVLASLYMLNPAVWINSAAWGQVDSLFMLF
VLLSLLRVEDKKFPAAAVYLGLAVLLKPQGLLFGPFFLLALLWRVRHKGFGRELAVSAAAGAAAFLLPALPFAWKMEPLW
LPKLYFGTLASYPYASLNAFNLLALLGGNFESVDGRLFSLTYGAWGTLLFVLSLLFVLVFFAKNMGRKGAGLYAGFLFAT
SAFMLMTKMHERYLFYGLLLLLAAYLRSKDRRLLVLFAGFSVTHFANVGYVLWKSVKGVFHIPKHDPVLMLVSLLNVILF
VMAWQAARTWAKKSAGADTRPQAGAAAGKGLFAGLMNRGSVAWDRDGAVAARQPLLTKKDGWLMGGITAVYAVIALVNLG
SFDAPQTFWKPASQTDTVVADLGATKQLTRIHDFAGAGEGEFRYEFSMDGVTWNDPIDVKSDYTKVFTWNVVKTDRNARY
VKITPLATGFALHEAAFFDGASDTPLPVSVLPGDDTQNSGEHRKIFDEQAIVPSAPGYKDGTYFDEIYHARTAYEHLHRI
EPYESTHPPLGKILISAGIALFGMNPFGWRIVGTLFGVAMLPLMYVMAKRLFGRTEYAAMATFLFAVDFMHFAQTRISTI
DVYGVFFIMLMFAFMYRYTTLSFYDVPLRKTLLPLGLAGVTFGIGAASKWIVIYGGAGLAVMLALSLWERFGEYRQARRS
LAEWSREASTAEGPDRVWNGGGPAGAADASGVSAVSGETGQPRIPGVFERFHTSESSDDDRERWTRIVRKFPRNTVYTLL
FCVVFYVLIPAAIYAASYIPFMLVPGPGHGFADVITYQQHMYSYHSGLVATHPFSSSWWEWPLMIRPIWYYKGEFVPPGS
MSSIVSFGNPLVWWPGLICVLLALWAAVKTKNKPLRFLLIAYASQYLPWMLVPRLTFIYHYFAMVPFMILMITFFLRHLI
ETRPKLKRWIYVYLAGAAVLFAVFYPLLSGLVVDDSYTGWLRWLPGWSFF
>lcl|NZ_CP026520.1_prot_WP_042229390.1_1347 [locus_tag=PC41400_RS07045] [protein=5'-3' exonuclease H3TH domain-containing protein] [protein_id=WP_042229390.1] [location=1555216..1556121] [gbkey=CDS]
MMGEIFVTQTRPAVLLVDGMALLFRAYYASAVTGYIKRTSAGVPTNAVYGFVRYFWDAVQTFSPSHVIVCWDMGSKTFRT
EQFTDYKANRAAAPEDLIPQFDLVKEVVDSFGVPNVGMSGYEADDCIGTLAQAYKDSADVFILTGDHDLLQLVDDSVSVI
IMKKGHGNYAVYTPDLLLEERNLSAAQIIDMKGLMGDTSDNYPGVKGIGEKTALKLIQQYGTIEGILENLSELPKGVRAK
IEAQLDMLHLSRELATIFLKVPVACALEECCWAIDHERVLTKFEELEFKGLAKLVSQPAFL
>lcl|NZ_CP026520.1_prot_WP_042229387.1_1348 [locus_tag=PC41400_RS07050] [protein=hypothetical protein] [protein_id=WP_042229387.1] [location=1556412..1556630] [gbkey=CDS]
MEANVLIIEKLENGELKTIDERNWNATMLAMMEHANFLYVGGKEYEMIEGRLDVENQKLEVLVLPINKQASE
>lcl|NZ_CP026520.1_prot_WP_042229384.1_1349 [locus_tag=PC41400_RS07055] [protein=hypothetical protein] [protein_id=WP_042229384.1] [location=1556656..1557618] [gbkey=CDS]
MNDNLNNNAGGLNDTEENANFNAAAGESASEAVAAEAAGAPEAGSPEAVLAARAADADKAAQARAARRTRTPKPLRVLNT
SLGVVLAANLLLVPAVYAESKAAEPQIIEWSNEDVKTYFDPSADWNIPLPEQLDKPGAGATPTPTPSAGVGTGGTGAGVG
TAAPIVYNNGLNWTDLMLYHLIFNSGGSYSSSGWSSSRPSYDYRTKKPYSVPTYDSTKFQNKQTANSTVKPKTSNTTGAF
NSKSTINSRNNSSKSSSTTGSSSTGKSSTSSGSSSTKSGTSSSSGSVSSGSSSKSSSSSSGSIGGKSGGFSSSGSSSSGS
>lcl|NZ_CP026520.1_prot_WP_042229383.1_1350 [locus_tag=PC41400_RS07060] [protein=glutathionylspermidine synthase family protein] [protein_id=WP_042229383.1] [location=1557618..1558823] [gbkey=CDS]
MDSRTFRVLGQPHKDREAYVAELGRMGFTWADLEQEQYWIDQLVLMDRGVYEELNQASRLLWDIFDKGARFVIGRRDLYD
MLSIPEVLWDGLDGLEPGPKGLLSRYARFDFSISQEGDIKLLELNADTPTGYVEASVVTPWMCSRYGGHSPNTAMPDLVR
QAWSEEMPEAAACIDYGKHLEDSGTIDMLIRHSGLEVRPVDCLDLWVDEGILKDGENREIRSMFALYPKEWMGVDDGGEA
LAYSIENGHLRLFNPLHAIILQSKGLQAVIWGLHEMMAPLFTPGEHRTIERYMLPTYNQPVLESSYVSKSMFGREGGSVR
MYGVSGEMDIQDEDGFDTSTLFTNVYQARADLPQISLTHGEYRLLTGMFVIDGQPCGLLGRAGGLITGNASHFVAIGVEQ
K
>lcl|NZ_CP026520.1_prot_WP_042229381.1_1351 [locus_tag=PC41400_RS07065] [protein=hypothetical protein] [protein_id=WP_042229381.1] [location=1558820..1559497] [gbkey=CDS]
MSRPAMNSRYGSGKKRSRVMIGIFSALALLLVFGLIGACSDKKNTSGGATASGGATTESSVVPWDYKVEQAKVGDLIGGD
MTLLPNNDMLPNDGNFATGDQVWVLSFMTAEMKTGDDGKNDVKLSGWTPLKTFKTESAAKTDMSQLKTEIQTEVDLIGVY
KTQYGDKHRDFAVVSMPSGHKVKQPISDERYTAIKDKKRVKIVLEEVHDFSNYDLAMSKFRGWAE
>lcl|NZ_CP026520.1_prot_WP_042229378.1_1352 [locus_tag=PC41400_RS07070] [protein=DUF350 domain-containing protein] [protein_id=WP_042229378.1] [location=1559494..1559883] [gbkey=CDS]
MIALNILISFIVIVALQVIGMVIFNWITPFNDMEELRKGNKAVGLVLGGKFFGTAIILGVAAYTNSSILHMAMWFAIGYV
CLLLTYFVFDLLTPGIKLSEQLKEGNVAVGIMLCAIYVGVAFSMSSLII
>lcl|NZ_CP026520.1_prot_WP_042229375.1_1353 [locus_tag=PC41400_RS07075] [protein=hypothetical protein] [protein_id=WP_042229375.1] [location=complement(1560024..1560287)] [gbkey=CDS]
MGDEFLKEESRGADLSTVESQRNDTIPEEFPEGPYGSSVLSESLGKSTPWRDDQRPTNRFRYENRELHYGIERDYPGDDQ
SEETPGP
>lcl|NZ_CP026520.1_prot_WP_042229373.1_1354 [locus_tag=PC41400_RS07080] [protein=DUF3055 domain-containing protein] [protein_id=WP_042229373.1] [location=complement(1560360..1560653)] [gbkey=CDS]
MSEFDFLYDHTEDTSTRFVCFVGSSLHRFDLAVTNTNRFYGKKMITDLQSGKTAIIGPDDLNEEGYLEHVYKISEEEAEE
LRSFLFQIVGTVNFTDI
>lcl|NZ_CP026520.1_prot_WP_241688809.1_1355 [locus_tag=PC41400_RS07085] [protein=protein phosphatase 2C domain-containing protein] [protein_id=WP_241688809.1] [location=1561140..1562144] [gbkey=CDS]
MYIGHRTGVTDPGTSRDPDKGPGFGERTKHEACLKKGRSGMPQLQCKVESVIRQGQGVCNEDALLLAPEEGLFGVIDGVS
GLTAYRDARGRTAGHIASALVAACFAGAGKADLERTTLEANARLHEEMERLQGQERSEDTGSRWGAVHAVVRVDEDRVAW
VQSGDCMVYAVYEDGVVRAVTRDSVEPHDERALALWRREYPGEWERRVQPPEVTEALRANRRRANRPGGYSVINGDDALA
RHLESGNIARSGLAHLLLVTDGIYPWREEHRHSTEEWVRAVIRTGLEAYVDELEAFEEDDPECRSVQRFKRSDDKTGILL
SFSKESRSDSKTVR
>lcl|NZ_CP026520.1_prot_WP_042229369.1_1356 [locus_tag=PC41400_RS07090] [protein=HRDC domain-containing protein] [protein_id=WP_042229369.1] [location=1562303..1563331] [gbkey=CDS]
MNVIYLTTLEKQMEEGKVRQAQFFAGEKTGIWQAMWNEVNDNGKPVQDIWYEGISWEELVHAIRKGLQLKLAEGYKPLLR
GIGEEQTGLSAKAEMTQLLQYYGELNADEAVYDELRAWRREQSFKEGRSPFILASNRMLQMIAAFCPQSLEELLHIPGFG
QNRASKYGGDVLRITRDKQRQTDFPLDWVAGATDRPQFEAWMNRQLERRVQKEAARQSGKRRLLEAMASGGSLAQIRTEL
SFTSRELVALLEELEREGYEVLAGIESELAKVPSEQLAEAERLFAEMGDRYLKPILQGLYGEQLPDAQTADGVYEWLRLL
RFKTRKMKEEAARERDSGQKAG
>lcl|NZ_CP026520.1_prot_WP_042229367.1_1357 [locus_tag=PC41400_RS07095] [protein=penicillin-binding transpeptidase domain-containing protein] [protein_id=WP_042229367.1] [location=1563550..1565550] [gbkey=CDS]
MKSPYTEQDPAKREADKRIHFTLRINIFFFATFALFCVLIVKLSYLQFVKGPELQIAKTNGMNTDTAIPPIRGNILSKDG
VPIATSNSTESLYFRVEGGTKNQDPIIETAKKLEKILGELGDKNLPKMTTEDIMKAMDLDFDINKQKIKPKTPYSEPRRI
KANLNKEEVAYLLSHRDELHGVEVVEESIRMYDQRKIAAQLVGYLKKYDDTANLNFYKSRKDQGYLLRETVGFDGLEMMY
QDELRGKPGKKSYPVNAAGKIVGNGTLEKPEKGQNLRLTIDTNVQLAAQQAIMDQLKWLHSPPSGYAKFAAPYARSGYAV
AMEVDTGKVVAMASMPDYDTNLWTGGMSTETYKQLEYFINNGTIRTSPANFPEKEQKKHPSSLVYLGSTIKPLSVLVGLQ
EGFLSPGSTYNDTGSFSFGKDKSTIRNSDGHSYGIMGPTKAIEVSSNTYMSAMIGIPFYNKYQKKSGELWDKHMAEFGLG
VLTGSGLPREIEGFSEIDAAAKTSSYQSAMVYASWGQNEKYTTLQLTQFAATLGSRGKRMKPQFVEEILDSDGKVVKGYE
PEILNQVDLPKTYWDTVINGMKSGAEGIDKLPYPVARKTGTSTQQVSGGTVDNAVFISFAPAQNPKLAIAVVVPEGGYGR
YGASPIAAKIYEAYDQYVGGLSTPKK
>lcl|NZ_CP026520.1_prot_WP_042229365.1_1358 [gene=corA] [locus_tag=PC41400_RS07100] [protein=magnesium/cobalt transporter CorA] [protein_id=WP_042229365.1] [location=complement(1565646..1566584)] [gbkey=CDS]
MKTRLVHKGAFTLVDDIHEALTPPESGFYWIDADVDDLEQLQPLFHLHDLAVEDCLSEEEQRPKIEIYDTHYFIVINSIR
FDDEEIFLRALNIFLGRHFIITVTRQKIQELRAIKPFLWDEEVNSPDRFLYHLVDLVVDNYVIVGDRIEDKIEKLEEDIL
MHTKKSHLNEIVGLRGEILWLKKMLVPQRDVVATLNRKELKLIDNQLQKYFGDIHENAVKIVDTFDTLRDLMGNLREAYQ
SSVANRANEIMRIFTALTTIFMPLTFITGIFGMNFTDILGLLTVPYADLFIFGFMGVLGLCMFLLFRKKGWI
>lcl|NZ_CP026520.1_prot_WP_042229361.1_1359 [gene=metA] [locus_tag=PC41400_RS07105] [protein=homoserine O-succinyltransferase] [protein_id=WP_042229361.1] [location=1567314..1568258] [gbkey=CDS]
MPIKIPDNLPAKEVLNQENIFAMDETVAYHQDIRPLRIAILNLMPTKETTETQLLRLLSNTALQVEFVLLHPKTHTSKNT
SPEHLEMFYKTFEDIKDERFDGMIITGAPVEQMDFEDVHYWDELKNIMDWSIDNVTSTLHICWASQAGLYHHFDVPKYTV
DQKVFGVFPHMITEENVKLLRGFDEFFFVPQSRHTEVRREDIENDERLVILSESEESGIYIVATKDGKQIFVTGHSEYDP
LSLKWEYDRDVNKGMDVAIPRNYYPNDDPSKKPMITWRAHANLLFSNWLNYYVYQETPYDLNAGRELSALSACL
>lcl|NZ_CP026520.1_prot_WP_042229360.1_1360 [locus_tag=PC41400_RS07110] [protein=aminotransferase class I/II-fold pyridoxal phosphate-dependent enzyme] [protein_id=WP_042229360.1] [location=1568297..1569466] [gbkey=CDS]
MNIESRLAQIGSVSEPTTGAVSFPVYQATAFRHPRLGESTGFDYARTKSPTRSVLEDAIADLESGDAGFACSSGMAALQT
IFTLFSQGDHLIVSLDLYGGTYRLLEQIMSRFGVSASYVDTNDVEALEEAYRPNTKAILIETPTNPLMMVTDLKLVCGWA
KNKQILSIVDNTLLTPYLQRPIELGADIVIHSASKYLGGHNDVLAGLIVTKGQELSEQMAFLHNSIGAVLGPQDSWLLMR
GMKTLALRMERHEHNATTIARYLQWHPLVTDVYYPALESHPGYAVQNRQSSGNTGIFSFKVTDARLVEPILRHVKLIAFA
ESLGGVESLMTYPAVQTHADIPLEIREKVGVDDRLLRFSVGIEHVDDLITDLAQALEHAQAEVEGDNQA
>lcl|NZ_CP026520.1_prot_WP_042229358.1_1361 [locus_tag=PC41400_RS07115] [protein=PLP-dependent transferase] [protein_id=WP_042229358.1] [location=1569463..1570620] [gbkey=CDS]
MSTDKKLGTKIIHFGGEIDKTTGASSVPIYQASTFHHFSLDNPPEFDYSRSGNPTRQALEDYIALLEGGARGFAFASGMA
AISSAFFLLSSGDHVICTEDVYGGTYRLLTAILTRMNIETTFVDMTDPEKVKAALKPNTKAVFLETPSNPTLKITDISVI
CAWAKENGLLTMLDNSFMTPYHQRPIEQGVDIVLHSATKFLSGHSDVLAGLAVVANESLGRRMKQIQNGLGNVLGPQDSW
LLMRGMKTLKVRMEGSEQNARRLAEWLDSHPAVEAVYYPGLPGHPGREIHEKQSSGYGAVVSFDVNGGERARKLLSEVKI
PLVAVSLGAVESILSYPAMMSHAAMSREVRLERGITDGLVRYSVGLEDIDDIIADLDAALSHLQG
>lcl|NZ_CP026520.1_prot_WP_042229357.1_1362 [gene=mqnC] [locus_tag=PC41400_RS07120] [protein=cyclic dehypoxanthinyl futalosine synthase] [protein_id=WP_042229357.1] [location=1570890..1572023] [gbkey=CDS]
MNPIDRILDKSLAGGRIDLEECVTLLESDQIEKMGHVANQIMLRHHPEPITTFVVGRNINYTNVCDVYCRFCAFYRAPGS
TEGYVLTNEDIFRKIQETLDVGGTEILMQGGTNPDLPFSYYTDLLREIKQRFDITMHSFSPAEIWKMKDVSNGLSLEEVI
RQLRDAGLDSLPGGGAEILDDRIRMKISKKKGSWRDWMDAMETAHRLGMNTSATMVYGFGETMEERAMHLLRVREAQDKC
IQNGYDSLGFLAFILWPVQPDNTNLRVEKTKPEEYLKMLAISRIMLDNIPNFQSSWVTMGPEIGRQSLHYGCNDFGSTMI
EENVVSAAGTTHKVNISSTLEMIRTAGKIPAQRNTKYEILRVYDENSVVEKDFIMQN
>lcl|NZ_CP026520.1_prot_WP_042229356.1_1363 [locus_tag=PC41400_RS07125] [protein=ATP-binding protein] [protein_id=WP_042229356.1] [location=complement(1572157..1573998)] [gbkey=CDS]
MKPNRSRKFGKIWGLLVRGIVRPVILCKQNVMRSLRMQLILAFALCIGVAIAAGGLVSELTKSWRTASSISYIQDKEITA
NQVRRLVERLEQIQKYGGDTDRYPAAKEESALKHLPSTDQELQSQKETEDTGTGIGSGPDRQADSGSSASSTAPKSAAPS
PMPESAVPDGTGTNGKAAPEPAGPGAAEADSKKTPTPGKAAEQPAPSFFPVNPADIQSLLDDEADRPGRKSLYIIDAAGK
VLYRTKGSGESRFDIPQLVGSAMDSEGTFRESHENREFTFMYPILLNGTQSFLVAKAVPEAQISYYEQDTSLPLIAGIAT
FLGLFYWITRRKMQLIEEMAQGLVQMSQGNLAHRVPERSHDELGVLAGHMNSMAFDLQHSLEEERKSQRLKNELITNVSH
DLRTPLTLIIGYLRLLKDKDYKDEEQAANYMQIAYGRAEKLKILIDELFEFSRMTNEGVPLAKQSVCVNDLIYQLSEEYI
TIAEQNGLTLRLVLPPMRLFVSMDPDQMIRVFENLLTNAVKYSPKPSTIEVIVQEEGRDVKVTVINEGDLSETEELDRLF
DRFYRMDAARSSDTGGSGLGLAIARSIVEAHDGRIWAENRGKSIMFHVKLRRY
>lcl|NZ_CP026520.1_prot_WP_042229355.1_1364 [locus_tag=PC41400_RS07130] [protein=response regulator transcription factor] [protein_id=WP_042229355.1] [location=complement(1573988..1574689)] [gbkey=CDS]
MGKETILVVDNEKEIVELIRIYLTNEGYETVSAGDGLEALDVLKSREVDLIILDIMMPRMDGVQACLEIRKEKNMPILML
SAKSEDMDKILGLTAGADDYLTKPFNPLELMARVKSQLRRYKRLNAPMPSETLSNRIELDDLIIDTATHEVHVEGRPVKL
TPREFSILELLGRNRGIVFSTERIYETVWNTPYFESDNTVMVHIRKIREKVESNPRQPKYIKTVWGVGYKIET
>lcl|NZ_CP026520.1_prot_WP_042229354.1_1365 [locus_tag=PC41400_RS07135] [protein=putative sporulation protein YtxC] [protein_id=WP_042229354.1] [location=1575037..1576011] [gbkey=CDS]
MSIVQFVFNNSAVFDISALQSRIESRLALLNQEGNPLRVVRDTLEQYTMLRIEGTIGGKKHQPKSEPVIEAAAAAAAEQI
VEGEEERLIRKLLVNTFHYKDLEEAAKIGRYCRQFLNGEEDGRSAAGEYGLDDYLDWGPIDGMKLRAGKISEPLKALLQE
GQSVNLPGFITFRLQEYMDELKEVVEYAIDEYVMDQQYQEFISLLQYFVYIQETKTAAAHLVHKEGAEFTILDEHMQPID
TSKMEASFTLEMIEKDISFEDMIVSTLISVSPATIYIHTREPDLPVIRTIMQIFESRATICSYCGICRTKLAGNSDSGSP
SPLT
>lcl|NZ_CP026520.1_prot_WP_042229352.1_1366 [gene=thrS] [locus_tag=PC41400_RS07140] [protein=threonine--tRNA ligase] [protein_id=WP_042229352.1] [location=1576376..1578310] [gbkey=CDS]
MIKLTFPDGSVKEYAPGTTVEQVAESISTGLKKNAVIGVVNGRQVDLNAPIEEDAELKIVTLDSEEGVEVLRHSTAHLMA
QAIKRLYGERAVKLGIGPVIEDGFYYDIDIENPLTPEDLTKIEKEMAKIVQENLPIVRRVVSREEAIAIFEELGDPLKLE
LIRDLPEDSVLTIYDQGEFFDLCRGPHLPSTGKIKAFKLLSVAGAYWRGDSKNKMLQRIYGTAFPKKAELDEHLHLLEEA
KKRDHRKLGRELKMFTFSKEVGQGLPIWLPAGAKVRRVMERYIVDLEERLGYQHVYTPVLANVELYKTSGHWDHYHEDMF
PKMVMDNEELVLRPMNCPHHMMVYKSDMRSYRELPVRIAELGMMHRYEMSGALTGLHRVRAMTLNDAHIFCRPDQIKEEF
ARVINLIRHVYEDFGIKDYRFRLSYRDPKDTEKYFPNDEMWEMSQRMLREVVEEMNLPFYEAEGEAAFYGPKLDVQIRTA
LKKEETLSTVQLDFLLPERFELEYIGEDGQKHRPVVIHRGIISTMERMTAFLLENYAGALPTWLMPVQAKVIPVSPSYEA
YANDVTEKLLAAGIRAEADNRNEKLGYKIREAQLEKTPYMLVVGENEMNSGTLSVRKRGEGDLGVHSLAHVVGLIQEDIA
NKSR
>lcl|NZ_CP026520.1_prot_WP_042229939.1_1367 [locus_tag=PC41400_RS07145] [protein=VOC family protein] [protein_id=WP_042229939.1] [location=1578548..1579408] [gbkey=CDS]
MTPVIDPRTEIGYVHIKVGSLSRSLPFYQDVIGLKLLKRDGAFAELTADGKKPLVILEELPGAAKAPKRGYAGLYHFAIL
LPGRKELGFALRSLIRSGIEVGQGDHLVSEALYLDDPDGNGIEIYADRPRETWRKDANGHYVMATDPVDWESLLELAGEE
PWSGMPAGTQIGHVHFHVTDLRAAEAFYCSLLGFEITLRFGPAALFVAAGGYHHHIGLNTWAGAGAPPAPANAAGIRWFT
VVLPDNGELERILARLREGSVDAVSREDGWLVTDPSGIGVMLTVRD
>lcl|NZ_CP026520.1_prot_WP_042229350.1_1368 [locus_tag=PC41400_RS07150] [protein=methyl-accepting chemotaxis protein] [protein_id=WP_042229350.1] [location=1579879..1581624] [gbkey=CDS]
MGKGKKKTGLTFFSKNLIFSISGIVLACMIMSVISYVVMSRALTGSLEVQAKGIATLWKNRFEVEDLKKGKATPSLDSTV
QKKLVSLLDELSKGNANVAQGFIFEPDVVEGAKTRVISNPTHLLEVGLKPGVLFDQPPEMVKAFQTLKQTKSPVIPPIYK
DEIGSWLTVLVPVLDEQQQLVAAFGIDVNASVISEQQYKLIWSLVTASMLFLLVGVTIQIFGLKKLLRPIQELISGVGKI
SKGDLTQKVVVRSNDELGQLSESFNEMVQSLHSTLEQVQVTVEQVASSSEQLSVSAEDSTKSSMQTTAIVQELATGTEIQ
AGNIDQTNRAINDISTRVGQIESNSRVVMDKVIIATNRATEGNQAIDMVVRQMNSISSTVNQSAESMKYLVDNALHIGRI
VEVINTIANQTNMLALNASIEASRAGEHGRGFAVVAGEVRKLAEQSSNSAQQITSYVESIQDGIQNAVQSMEQGTKEVNE
GLHLVNQAGSSFNEIRNAVDEVSLNIEVVSTALEEMTSRVSEVLTSMNRILDVTNEVVGGTQNIAASTEQQLASMEEITS
SANVLSSMAEELQEKTTQFKL
>lcl|NZ_CP026520.1_prot_WP_042229348.1_1369 [locus_tag=PC41400_RS07155] [protein=mechanosensitive ion channel] [protein_id=WP_042229348.1] [location=complement(1581802..1583298)] [gbkey=CDS]
MDVRNWGEWNFLMMNGNSIWRLIGAVLVLIIGLWIAGAASKAVYKGLLKANVDTRGRKFFPPDSSVSRIVSQVVKYFIIL
VTLLVVLELLNFTNVLNPFVGFIAIITSYIPHLLAAALLAAVAHILGSMLKSVVSGFFGSHTIKSKMPSLVHYREVAGNI
AYAIVILLFAPSILASLQIPAISVPISAVINMVIGYLPLLAGAILLIVLGHYVAKFVAKLASTLTSSLNLQRWIGAGMNV
SQFVYSIVYFLILFPIAVQALNLLQIESIRVPAQGILNLVMAWIPKILIAAFLLYIGYILAKIVRNLIITLLSPMDIDSK
LNGLMPMFRRKAADAGAEMGVPGSVNPAASVSALPITKWIATLIAVFVFGFFLVESTNVLDLEFISATFAALMVMLPKFI
LIVIIVLIGIILAGFAERMIRPGNPLKGFVQPVIIVLAVVIGLTEIGLGSVIITSGYMVVMVALGAAFIISVGIGSIPAV
KRFWDKKQNSKDHDNISM
>lcl|NZ_CP026520.1_prot_WP_042229347.1_1370 [locus_tag=PC41400_RS07160] [protein=3D domain-containing protein] [protein_id=WP_042229347.1] [location=1583512..1584213] [gbkey=CDS]
MLPDSSNPTKWFSMVMMLNIALVLCGQDQTSLTGLTDTAALEKKIAPAISGLASRQTSEAPEKSVQSQKGTADAQAASGQ
SRKVSYKLMPDSSQDLSQYKAVEVVATGYYSGIESTGKNPGHPDYGITKSGIKVKRDEHSLSTIAADPRVFPIGTVLYIP
GYGYGIVADTGGAIKGKKIDLYYETKDQIYKEWGKKTVNVFVVKEGTGKVTQMIWNQLVQEIFKNNPVDTPEA
>lcl|NZ_CP026520.1_prot_WP_042229346.1_1371 [gene=liaF] [locus_tag=PC41400_RS07165] [protein=cell wall-active antibiotics response protein LiaF] [protein_id=WP_042229346.1] [location=1584390..1585418] [gbkey=CDS]
MNGSIWGRLAGGFLLIAIGLLFFLKQLGFVTLSIGALISTFWPLFLIYAGLMSVLFNRGRQDGAWVLILIGGIFQLRNLG
WAPYSLGEMFQFLIPVVLIIFGIKMIFGRKSSRRHDSEYGGSRDGHGHGDGNGHWGSDWERKMEDKWEEKARKWQEKISR
REARRGRKADWKNDGVYPEAGEASRESGYAPPPPPPPPAGGSYKPGPDYHYVNSYAPPQNKSGFIGDVYMGQDYWELTPL
NISHFIGDTSLDLTKASIPMGETRINISAFIGDVKVFIPSDLDVEISVTASSFVGDMNVLERREDGIMRSMNYRTPYYQE
GAKKINLHVSMFIGDIVVKKMG
>lcl|NZ_CP026520.1_prot_WP_042229344.1_1372 [locus_tag=PC41400_RS07170] [protein=sensor histidine kinase] [protein_id=WP_042229344.1] [location=1585431..1586468] [gbkey=CDS]
MIKLWSMKWQLMLYFLASSLLTLGILGFAQFYMPWDTAGGGWKFGLAVFAVLLGQAVGFVASRTFQRKVDVLHIAMLQLA
KGNLKDRIQIDPIDSFHPVYKSFNEMAGALEERTRLLQEMGEERSVMERGSVETAVVEERRRLARDLHDTVSQELFAIHM
SASSLPKVIGHNPDVAGQLMEQLIQMSHHAQKQMRGLISQLRPIELDGSTLEQALEKWFPEYCRAHELQGRLDVETEQPI
SEAIEHQLFLIIQEGMANVVKHASATEIELQLHDSGRQYVLQLADNGKGFERSAISSASHGLSTMRERAQKLGGDAEVLS
GSGSGTTVRVRIPKFVKEPKGEASL
>lcl|NZ_CP026520.1_prot_WP_042229342.1_1373 [locus_tag=PC41400_RS07175] [protein=response regulator transcription factor] [protein_id=WP_042229342.1] [location=1586465..1587127] [gbkey=CDS]
MKEQIKVMVVDDHDMVRVGLRTYIMLDEELSVVEEASNGEEALRKLGALDAERIPDVILMDLTMPVMNGIEATREITQRF
PSIKVIMLTSFLEEDKVVQAVEAGAISYVLKTISSEQLAHAVKGAAAGMPVLNPEVSLALTRGLRQRSSQTEDEVLTARE
REVLSLISDGKSNKEIAEALHISIKTVKTHVSNLLMKCELDDRTQLAIYAHRKGWAGTGA
>lcl|NZ_CP026520.1_prot_WP_042229341.1_1374 [locus_tag=PC41400_RS07180] [protein=YugN family protein] [protein_id=WP_042229341.1] [location=1587243..1587656] [gbkey=CDS]
MKALNSSLQETVEAFDNVRDYLHTFEFTLGGNWEYDHGSFDRYLDEAHKVWLRIPFEVTEGTLDGDTDETDAMVKIGAPY
VLKHVYNEGNDPEAHMMTVSALVNQFQTPLDPDAEIESEWIGKAEKLMRDVEQKWLH
>lcl|NZ_CP026520.1_prot_WP_042229338.1_1375 [locus_tag=PC41400_RS07185] [protein=trypsin-like peptidase domain-containing protein] [protein_id=WP_042229338.1] [location=1587889..1589499] [gbkey=CDS]
MDDHKKEFGPDRSNDSDLNRDRDRERELERREAGYEPGYDSRREEAAAQEKPSYYYSYGPHKTGMNEETPSTESDTTAYS
NHEGSAPVEVTPPKPVRPFSFSGDGAPKQGGGWNGSGPAKKRSSLRSGFAAFLAGAVVVSTLMFAADKTNLFTGKQPMTL
PGAQPNSSQTVAAGSASKVSLDSARPDNIAQIFAQASPATVKIESYVKTKSSSQKSQMNDPFYRFFFGDQGDSGKQDSGS
GKSDGTLQLAGSGTGFIFEKSGYILTNEHVIDGADEIRVTVQGYDKPFVAKLLGNSYDLDLAAMKIEGDKDFPILPIGKA
DSLNVGDWVVAIGNPYGFDHTVTVGVLSAKERPITIQDEQGTRNYKHLLQTDASINPGNSGGPLLNMNGEVIGINTAVSS
QAQGIGFAISESTFSTVLDNLKNNVTIPKEKAPYIGVSLQNVDKEWVNELKLPNTEGAIVAEVTRKSPAFQAGMRPYDVI
VDVNDQKVTNSQDVTTKIQALKVGDKAKIGVMRDGKRVEINVTVGDKNAADTTEKK
>lcl|NZ_CP026520.1_prot_WP_042229337.1_1376 [locus_tag=PC41400_RS07190] [protein=response regulator transcription factor] [protein_id=WP_042229337.1] [location=1589699..1590388] [gbkey=CDS]
MRETILVIDDDEKITSMLRRSLAFEGYTIVTANNGNDGLKKMLEQEPHLVILDVMMPVVDGWEVCRRIREGGSSVPILML
TAKDEVTDRVKGLDLGADDYLIKPFALEELLARVRVLLRRRNERAEQPTNRLHYDNITLDMDTREVFRESQLIELTTKEF
DLLHLFMQNPKRVLSRDIIMEKIWGYDYSGESNVLEVYIALLRQKTEEYGHKRIIQTVRGAGYVLRGEN
>lcl|NZ_CP026520.1_prot_WP_042229336.1_1377 [locus_tag=PC41400_RS07195] [protein=HAMP domain-containing sensor histidine kinase] [protein_id=WP_042229336.1] [location=1590389..1591909] [gbkey=CDS]
MSIRLRLTLWYSGILSVTLLLFGFGLYLFLSFYIYNDLEKQLGRTWTETYERIKPLAAPSIRGRVPIKLELDNVDMFSEN
TLLQLASLNDTEGPVIHDRSSVLKNLELTLPITNNAAQKLRGKDSLVVERLNLKVMDKSYPIILVSRGFDITDPVTGNKL
TRGVLQVGATINYYENFFNLLQYTLAILAMLTVLIAGTLGWFLARKALRPIEQVIVATQQIEKGTDLDKRILYDGPEDEI
GRLTVTINGMLGRLQLVYSELEEAYRAQRRFVSDASHELRTPLTTIRGNVELLEKMWSRTLTSGQAGGQDKEQMEMSIEA
MQDIAGEAHRMSRLVNDMLSLARADAGFEMQRQPLELRPLVEEVARRAQFLPRQVQWEVGDLEPLTDVVVVGNKDYVQQL
LFIFIENAFKYTEEGSVRLDALRSGDQVGLRIADTGIGMDKEEVPHIFDRFYRADVSRGRTAGTGLGLSIAKWIIDEHKG
SIEVKTRQGEGSTFIIWLPVVFLPVS
>lcl|NZ_CP026520.1_prot_WP_042229335.1_1378 [locus_tag=PC41400_RS07200] [protein=4-hydroxy-3-methylbut-2-enyl diphosphate reductase] [protein_id=WP_042229335.1] [location=1591955..1592908] [gbkey=CDS]
MKVVKISPRGYCYGVVDAMALAMQTAKNLDLPRPIYILGMIVHNSHVTRFFEDEGVVTLDGPNRLEILEQVTSGTVIFTA
HGVSPEVRRKAREKGLTVVDATCPDVTKTHDLIREKTAEGYEIIYIGKKGHPEPEGAVGIAPEHVHLIETLADAEELNIA
SSKILITNQTTMSQWDIKHIMKKLLESFPTAEIHNEICLATQVRQEAVAEQAKETDLVIVVGDPKSNNSNRLAQVSEEIS
GVKAYRVSDVSEIDKSWLMGVRTVGVTSGASTPTPITKEAIAYLEQFDPAHPATWDITRTVNMKKLIPMAKTKAAAE
>lcl|NZ_CP026520.1_prot_WP_042229333.1_1379 [locus_tag=PC41400_RS07205] [protein=DUF2062 domain-containing protein] [protein_id=WP_042229333.1] [location=1592941..1593492] [gbkey=CDS]
MQISEGALRKKFDLKQMGRWFKYKYLLLLRAKGGPSMVATGFSIGLAVEMFTLPTVGLAFFLIFPLVWLLRGNFAGALIG
FVFGKIIYVPMAILLNKPVGSLVVPPGFKEYLIHHVPAFLANVLKGSLELIVGGMIVGGILGLIMYFPIMLLLRLHANRR
KDRRRLRKEHQHAMASGLPGDAD
>lcl|NZ_CP026520.1_prot_WP_042229331.1_1380 [gene=aroF] [locus_tag=PC41400_RS07210] [protein=3-deoxy-7-phosphoheptulonate synthase] [protein_id=WP_042229331.1] [location=1593692..1594735] [gbkey=CDS]
MIVILSNKVEEARIQEIVHIIEKHEVKAHVSHGEDRTVIGIIGQADPVLAEQLRQLKGVEQVVKISKSYKLASRDFHPAD
TVIRIKDVEIGGDQLVVMGGPCAVETPEQIDEIARLVKAAGGQVLRGGAFKPRTGPYSFQGVGVEGLVMMAEAGRKHGLL
TITEVMTPEYVDVCAEYADILQVGTRNMQNFDLLRKLGSIQTPVLLKRGFSSTYDEFLNAAEYILAGGNPNVMLCERGIR
TFETYTRNTLDLSAIPVLQSLSHLPVISDPSHGTGRRELVEPMTKASVAAGANGLIIEMHTDPDNSMTGDGVQSLFPDQF
AKLLQDLEKLAPLCGKRFDTAKTPALV
>lcl|NZ_CP026520.1_prot_WP_042229330.1_1381 [gene=glnA] [locus_tag=PC41400_RS07215] [protein=type I glutamate--ammonia ligase] [protein_id=WP_042229330.1] [location=1594970..1596394] [gbkey=CDS]
MSVQDVLNIIKEKNIEWIDFRFVDLLGKAHHISLPACEVEEETFENGVAFDGSSIHGFRGIEESDMVMMPDTESVYVDPF
TAHPTLIVMCNIHTPDGERYERDPRSIAQKAEEFLQTAGVGTTANFAPESEFFIFDDVRYESGMNTSYYSVDSEEGAWNT
GRKEEGGNLGFKIGVKGGYVPVQPVDTQQDIRSEMCRLLQEAGLRIERHHHEVATAGQAEINFRFDTLTKTADNLMKYKY
IIHNAARQYGKVATFMPKPLFGDNGSGMHVHQSIFDGDTPLFYDKNGYANLSEMALHYIGGILYHAPALLALTNPSTNSF
KRLVPGYEAPVNLVFSKGNRSAAVRIPVAAVTPKGCRIEFRTPDSTANPYLAFAAMLMAGLDGIKKKIDPRALGYGPLDK
NIYELSDEDKSEIRSVPGTLDEALDALEADHEFLLEGDVFTQDFINNYVDIKRGEAKDVAIRIHPHEYSLYFGC
>lcl|NZ_CP026520.1_prot_WP_042229328.1_1382 [locus_tag=PC41400_RS07220] [protein=DUF1800 domain-containing protein] [protein_id=WP_042229328.1] [location=1596558..1597868] [gbkey=CDS]
MKPWTEKEVLHLLGRASFSATPGEVDRCLKLGREETVRRLTSGQPLADGAASALVFSEVKADEKPLIVTGMGDQQTYWLY
RMIASPAPLVEKMTLFWHGHFASSQYKVSDIKLMVRQNGLFRQHALGSFRKLTEEIGQDPAMLLYLDSYTNQKGSPNENY
AREMMELFTLGIGNYTEQDVKEAARSLTGWSYTRKTDIVQYQEARHDAGHKVFLGEAGYFDSADVVRVVFKQAALPKFLA
TKLLQYFAAENPSEAWIDRVAGNIAKMPTIENVLYELFVSDEFYEEAYRMSLIKNPAEYVAGAMRTLGLPLTATFRASMG
LMGMDLYAPPDVSGWKGGASWLATSRLIARYQFAEKVAKRVTDDQLQSLVPQGTQTGPEAYVEGWRRGMGLAPLGKNTMN
VLKQYAAETIVNSKKPAGGKRSLLQLLLISPEAQMK
>lcl|NZ_CP026520.1_prot_WP_042229327.1_1383 [locus_tag=PC41400_RS07225] [protein=DUF1501 domain-containing protein] [protein_id=WP_042229327.1] [location=1597885..1599114] [gbkey=CDS]
MNLTRREFLIKGTALIAAIGIGGPLLYADSKNILKPGPPAGAKDRVLVVVQLSGGNDGLNTLIPYGIGAYHDARPRLRLQ
QHEVLDINGQLGLHPSLAQLHGLYQRGKVAIVQGVGYPKPDHSHFRSMEIWHTGEPEKRISDGWLGRYAFSSLDTANPLR
AVQLGKTETRALRHDSISLPVVSALDSYKIFTSKPPDPDRGRLNKAFLDMYNLNKQTAPLRVACSSGIEAYRSVEAIGSL
RPGYTAGAPYPDTSLAKDLQLAAQLMASGSGTRVFYTQLGTFDDHANEKAHHASMLSTLDEGLGAFYRDLEAHGLQDQVM
VMAFSEFGRRTKENGSGGTDHGTAGPVFLLGGGVSGGLYGAYPSLTKLDKGDLKAELDFRSVYYTVVESWLNGDARSAVG
RLYEKLSFV
>lcl|NZ_CP026520.1_prot_WP_206735006.1_1384 [locus_tag=PC41400_RS30060] [protein=YjcZ family sporulation protein] [protein_id=WP_206735006.1] [location=complement(1599226..1599321)] [gbkey=CDS]
MSGGYGGVFTSTAVILVLFILLVIVSRTFLY
>lcl|NZ_CP026520.1_prot_WP_042229324.1_1385 [locus_tag=PC41400_RS07230] [protein=YheC/YheD family protein] [protein_id=WP_042229324.1] [location=1599548..1600354] [gbkey=CDS]
MDSQNVLPNTKLGKNQFLNRHSELKKFQPDTAQATKDNLYDFLKRYRMVYVKPNNGTGGKGIFKVEYSSKGGYRYQINET
KKKFETYDELYRTMSKRFGDTKHLVQRGIHLLTSQGRPFDIRIMVQKNLRGDWENTGIIGRVAHPDKIVTNYHRGGTPTE
ISKLLKEYMKKEQVPDYVAKLNRLGHEIAEYMENGYPKVTAIGVDIALDNVLKPWILEVNTKPDASIFKKLDDKSIYRKI
WKYKKANRILVRRKNLGNSSTSKPLTRV
>lcl|NZ_CP026520.1_prot_WP_042229323.1_1386 [locus_tag=PC41400_RS07235] [protein=YheC/YheD family protein] [protein_id=WP_042229323.1] [location=complement(1600503..1601630)] [gbkey=CDS]
MAKPVIGILTWREGHSFKEPGYFRSLVREGQKLGATVFIFSHKDAFPDKKQVRGYVPDGNTWTSRIFPWPDVVIDRCRTA
QEGYSEFRKQKHFVYANHTFTNKMSATRLFSNEEGLKKWIPATVDYNAQNLVRMFKNHRIVYIKPGNGTGGKSIVKISRV
KDGYRLLGRTKGYAKRNVIFRNLTSLTNWLNNWVRQERIRKGSFMIQQGLDLELLPGHVSDMRLLIQKNDQGKWEVTGEG
MRIGGVNNPTSNLHGGGRASTTEAVLHKRFTPEKTREIIAECHNLAHGVVTVIEKHFGRMLEYGLDIGIDKQGSVWLIEV
NPKPGRELFRKMGRRDLYQQSIRRPLQYAIYLNEQEQKQEQKQEQKQEQKQEQEK
>lcl|NZ_CP026520.1_prot_WP_042229322.1_1387 [locus_tag=PC41400_RS07240] [protein=SH3 domain-containing C40 family peptidase] [protein_id=WP_042229322.1] [location=complement(1601845..1602630)] [gbkey=CDS]
MKKIAKLVLGLSLLAGSSLSAGAAFAATAPAITTSSTIITTGNPYLKQLPYSSQGKIYGTVPKGTKLTVLENTNKYYVKV
NYNGQTGYISTLHLKAVGGGTSTAPAPAPAPAPSKPGTSGSSSATWSKQADSILSFAKSLQGKVQYAYAKNDTKNLIFDC
SSFTKYVFNKEGIHLPWGARAQYKLGTPVSKSELRKGDLVFFSTTATAGNKDTVNKIGHVGIYIGNNKFIHNVNPKDDVV
IGDLNNTWQKNHYVASARILK
>lcl|NZ_CP026520.1_prot_WP_042229320.1_1388 [locus_tag=PC41400_RS07245] [protein=C40 family peptidase] [protein_id=WP_042229320.1] [location=complement(1602667..1603191)] [gbkey=CDS]
MKKQLGKKLIIASLSAAIALSGGLTALPQQSHAAVSSSAFKAKADKIITLGNKYLGRPYKFGAKTGNTSSFDCSSFVQYV
YQQNGIKLARGARDQSKNGVKIKRSELQPGDLVFFSTTKTMKYSASSVNRIGHVGIYAGNNKVLHTFGKGGVTYSNMGSG
WWDNHYVTAVRVIK
>lcl|NZ_CP026520.1_prot_WP_042229318.1_1389 [locus_tag=PC41400_RS07250] [protein=YheC/YheD family protein] [protein_id=WP_042229318.1] [location=complement(1603412..1604761)] [gbkey=CDS]
MAKQVPSRKRRQASPLPAGLLRVDPRSSGSKTPPPGGSTGPEEATLPILDAAENPQRRSAETGRIDGLTEEELLEAQIHA
NAGRSNPLIGVLVKTLQPSSPNLFGGITTFCKELDEASKEQGARVFFFSVDGIKPGSPQITGWHYTSGDWKKGTFPMPSV
IYNRLTSRILENRPKVQQFFRDAKASYGAKVFNEKYLNKTEVFQALKDDAACKSFLPESYLFKNYAMLQSMCKKYDTVFL
KPVLGSLGKGILRVSVLEGGAYSCEIPTLNGTSRKVYPSLLQLFNAQAGKLKTSRYQIQQGLQLVQVGGRPVDFRALVQK
DQQGEWAITSVVARIAGANHFVSNLARGGSLSTVKEALAQTSLSAADHKKISTQIRKAALAVAEGIQTHIPALFGELGVD
LAVDTGGRVWLLEINSKPSKNDNTPLDGGNKIRPSVRRVIRYSKYLANH
>lcl|NZ_CP026520.1_prot_WP_164977110.1_1390 [locus_tag=PC41400_RS29665] [protein=hypothetical protein] [protein_id=WP_164977110.1] [location=complement(1604845..1604991)] [gbkey=CDS]
MLLWVDKRIYAVSGWHKEKTFTLWLGIVQLALGLLSLAVHYFLQYSVW
>lcl|NZ_CP026520.1_prot_WP_042229314.1_1391 [locus_tag=PC41400_RS07255] [protein=Ger(x)C family spore germination protein] [protein_id=WP_042229314.1] [location=complement(1605055..1606287)] [gbkey=CDS]
MHPIRARLPLLLLTVPLLSLLLAGCWDRQEIEDRALILGMAIDRAENPANMQSKDQVNHLEPNPIPADKMIRVTASIAVP
GRIPLGPGEGGGGSGGQDTTKPVWVVSVFGHTVNDALNNLQQQIADPKFLIHLRVIIVSEEVAYDTLEHINDYFRRDSEV
RRRTWLLISKGEAARFLNVAPPLERVPTLYVLAMLDKSVQLGKFPRDYIGSFWSTQSKQGQEGYLPYVDIRRNENILIGG
MALFRNNKMVGTSEPLEIGAFMAVKEMNPAGYSPLVDVPGIGPVMATTTLRKSRIDLEMKNGSPRFKVKIYLEQDLQEKY
FEDSVIDEKAISKIEETISKEANKIILNFIRKTQRMKSDVFGFGEHIRAKEHAYWSARIGSKEKWETHYAVLPVEVQTTV
RINRTGMQFR
>lcl|NZ_CP026520.1_prot_WP_042229937.1_1392 [locus_tag=PC41400_RS07260] [protein=endospore germination permease] [protein_id=WP_042229937.1] [location=complement(1606268..1607374)] [gbkey=CDS]
MNDQRLTNWQAMCILIGTIAGVGVLPLPRLAVTAGESAAPLVTVGGILLGALLVMLWSLLGMRFPDLSPVAYSKLILGKW
ASGVLLFVIALFYLVATAFSFREFSEVTGTFVLRRTPLEVILFMMLLLVVISARKDIYSFTLIHAFYTPFVVGPVVFMMI
LSFSEVELLNLQPFTTNPDFTFWVGSLGITTLFRNVVIYTFIVPRMRNPQKALKVGLTGTAISGGLYLVIVITTVGVFGA
EEIKRLVWPTLELGRATIIPGEFLERMDALFLVTWIISVFNSLLSGYYLILLIIKEKLGLKDHRMLATFLMPFMFLAAMM
PQNVIQLYRLSTHILPWGLIAMIIIPSALLVVAGIRGKGGVAHAPDKG
>lcl|NZ_CP026520.1_prot_WP_053228699.1_1393 [locus_tag=PC41400_RS07265] [protein=spore germination protein] [protein_id=WP_053228699.1] [location=complement(1607367..1609031)] [gbkey=CDS]
MMWNWMKNRKTGGPLSGSPMPEPRIVVPSLYIEDILSGKQLSTGIEENLKLLQDVLQYNDDLTVRRFQLFEGTSAALVFF
SSFVDQEMIRSYLIKPLMTHTEQEDLIVDAESLASYLTEKVVQVSQITEEKAPDRIIDELTMGVTVLLVDGIAAAFLADT
RKIEKRSIEQPQTEQVIRGPREGFIEHLQTNLALIRYRLQTPDFRVHTMHIGRVTKSKVALCYIQGITHPALVHEAKKRL
SAIDTDSILDAGYIEQFIEDHPLSPFPQIQNTERPDKTVAALVEGKFVIMVDGSPFALIAPTVFSQFYQTVDDYTERFLI
GSMIRFIRLIALLFSLVFPSLYVAVISFNPEVIPTDFAVAVAGSRAGVPYPAMIEVLLMEISMEVLREATIRLPQLIGGA
LSIVGVLVIGQAAVSAGFSSPITVVIIAMTTVGSFATPAYNAAIALRMLRFPLIMLAGSFGLYGIMFGMILITNHLLSIR
SFGVPYMTPITPFNAQEWKDTLLRGPIWWMSDRPKHLKSLRTKRRKHTDRYYRSPSYSWDKVPEVPDQGGAPDE
>lcl|NZ_CP026520.1_prot_WP_241688810.1_1394 [locus_tag=PC41400_RS07275] [protein=hypothetical protein] [protein_id=WP_241688810.1] [location=complement(1609220..1610368)] [gbkey=CDS]
MRKRLMNTARLLMMICAILLVPSNAFAAYTIGTETPFPTPSAKESYTSGDVIFSKTGHPVFVSNAPESIYDHQVNKTLYR
DTMNGTFRFWSSHRNMTTGNIQFFLYVKNPSATNSVKLYLIKDGYGDSNAGGDVTKAASTATTCFMSKNTGGTYLATIAP
GQGYAYEYSAAGGISSQKAIVHIADFRAVDVTTGSDADVKISDLSTNTGTTNLAAYANTATIASTNYNAKTKDDYRGLLQ
YSGRTADINLTLTSAAPAKFITQSDDGGYTNEQDILLSRWNVDGNPQDQAPASFNGKIRGAYWCTDYTYRINITNSSGLP
NVYTLYGSRTSGAFITYKISTVTNRHVDKGNGIVINDTPTYTLHTIVQPSNNLPLSLDFVAQ
>lcl|NZ_CP026520.1_prot_WP_042229311.1_1395 [locus_tag=PC41400_RS07280] [protein=nitric oxide synthase oxygenase] [protein_id=WP_042229311.1] [location=complement(1610692..1611789)] [gbkey=CDS]
MNSTNDIADKARAFLYTCYSEWGRSEREAEERWLDVYAQIEETGTYVHRPEELEYGAKLAWRNSNRCIGRLFWNTLSVFD
GRHIGTFEEVRDALLTHIRFATNDGKIRPAITVLPPAGKDGNCPVRIWNHQLIRYAGYETENGPVGDPNSIPFTQICREL
GWKGEGTHFDLLPLVIQIGDDAPRWFELPRDLVLEVPLRHPELPGFEELQLKWYAVPIIADMRLEIGGISYPAAPFNGWY
METEIGARNLADEQRYNLLPRTASLMGLDTGSNATFWKDRALIELNAAVYHSYRNSGVSIVDHHTAAQQFQLFEQQEAQA
GREITGDWTWLIPPVSPATTHLFHSKYENTWKTPNFYHQDKPYEK
>lcl|NZ_CP026520.1_prot_WP_042229933.1_1396 [gene=thpR] [locus_tag=PC41400_RS07285] [protein=RNA 2',3'-cyclic phosphodiesterase] [protein_id=WP_042229933.1] [location=complement(1612068..1612640)] [gbkey=CDS]
MTDALERLFIAVPLSGPVRENLSGWTGRLRESGTPFHKWVHPEDFHITLKFIGDTSRSTLPALSPLLEQTAEAHSPLELT
AAGAGTFGPEDAPRILWAGVQGDLDALHRLQRDVEAACSSLGFEPEDRPYRPHITLARRYTGGRKWARERLDAPGEPPSA
SWTAEEIVVYRTHMGRSPMYERAGAFALRG
>lcl|NZ_CP026520.1_prot_WP_042229309.1_1397 [gene=serC] [locus_tag=PC41400_RS07290] [protein=3-phosphoserine/phosphohydroxythreonine transaminase] [protein_id=WP_042229309.1] [location=1612780..1613868] [gbkey=CDS]
MGNRAYNFNAGPAALPLEVLEQAQKEFVDFRGIGMSIMEISHRSAEYEAVNNETQNLLKELFSIPDGYKVLFLQGGASTQ
FTMIPMNFLQAGQTAAYVQSGAWADKAIKEAKLFGETAVVASSKDQKFMRMPDLGALNLSDQTAYLHVTSNETIEGSQVQ
SYPDTGSIPLIADMSSDILSRPVDVSKFGLIYAGAQKNLGPSGVTVVILREDMLERANANLPTMFRYDTHVKNDSLYNTP
PSYSVYMINLVLHWIKRQGGLAALEKINRDKTDLIYNAIDASGGFYQGCVDADSRSRMNITFRLQSEELEKLFVKESQAN
GFVGLKGHRSVGGLRASTYNAVPYESCQALAQFMNDFSKRNG
>lcl|NZ_CP026520.1_prot_WP_009676211.1_1398 [locus_tag=PC41400_RS07295] [protein=response regulator transcription factor] [protein_id=WP_009676211.1] [location=complement(1614146..1614859)] [gbkey=CDS]
MKKKVLVVDDEPSISMLIEFNLKLVGFEVQCVYDGEAVFDVIGQFRPDIIVLDLMLPKMDGLQVCRKLRSQNNLVPIIML
TAMQDLSDKIAGLDNGADDYMTKPFSPQELISRIQAIIRRIQTLPASGEEAPITIGQLSIRPDQREVTCAGKPVDLTPKE
FELLVFLCKHRGKVLSRQQLLHGVWDYHFLGDTRIVDVHISHLRDKIEANARTPEYIMTIRNVGYKLTEPAIKESFA
>lcl|NZ_CP026520.1_prot_WP_042229308.1_1399 [gene=trmL] [locus_tag=PC41400_RS07300] [protein=tRNA (uridine(34)/cytosine(34)/5-carboxymethylaminomethyluridine(34)-2'-O)-methyltransferase TrmL] [protein_id=WP_042229308.1] [location=1615001..1615465] [gbkey=CDS]
MAFHIVLVEPEIPANTGNISRTCAATGTYLHLVRPLGFSTDDKTLKRAGLDYWHSVKLEYHDSFDEVKEKYAGSRFFFAT
TKTDKRYTDFEFRDGDFFVFGRETKGLPPEILEAHPGQTMRMPMSDAVRSLNLSNSAAIILFEALRQNDFPGLR
>lcl|NZ_CP026520.1_prot_WP_042229307.1_1400 [locus_tag=PC41400_RS07305] [protein=hypothetical protein] [protein_id=WP_042229307.1] [location=complement(1615640..1616002)] [gbkey=CDS]
MKTVSQEIQDLSVKSLESTFNKLTNAYKSMTEKGSNTTLVQKRQNAVKIGLESLKGIWNGEDFSYDAEIILTSKEVLQSI
IPSVEQQIAKAKEGSSQKTLNTHRLTALKLAIESLENRLI
>lcl|NZ_CP026520.1_prot_WP_170318711.1_1401 [locus_tag=PC41400_RS07310] [protein=AbrB/MazE/SpoVT family DNA-binding domain-containing protein] [protein_id=WP_170318711.1] [location=1616688..1616945] [gbkey=CDS]
MKPAGVVRKVDQLGRIVLPKSLRKRYQMNEGDPVEILVQGDHIILERYRPKCVFCGEMDHVNEFKERHICSNCLGEMNAM
KQPRV
>lcl|NZ_CP026520.1_prot_WP_042229304.1_1402 [locus_tag=PC41400_RS07315] [protein=DUF2500 domain-containing protein] [protein_id=WP_042229304.1] [location=1617120..1617536] [gbkey=CDS]
MININGIGFGDGPGMFGLVSALFPLVFIAILAFIAFTLVKSILQWNHNNKQPILTVEARVATKRTAVSHHHSASNESTHS
HSTDTSYFVTFEVESGDRIEFRISGSEYGQLVEGDMGKLNFQGTRYLGFARHRVVSYP
>lcl|NZ_CP026520.1_prot_WP_042229302.1_1403 [locus_tag=PC41400_RS07320] [protein=serine hydrolase] [protein_id=WP_042229302.1] [location=complement(1617596..1618612)] [gbkey=CDS]
MKEIISESMQALKEATGFSGTVLAGNKEEILAAFSGGYANRSEGSLNRADTRFGIASGCKLFTATAICQLVEKGQLSFDS
KLRDCLPLEFKNFDQEITIHHLLTHTSGIPDYFDEETMDNFEDLWTDYPMYRVRRPQDFLPLFAEKKMKHETGRVFHYNN
GGYIVLGLVVEHVSGLAFSDYVEQFIFAKANMKDSGYFEADALPERTAAGYIDHEDGSWKTNVFSLPAKGGPDGGAYVTA
TDMMKFWESLMTFELLSEKTTSILLQPHVHVTNDIYYGYGLRMKLRQQDVEKYILMGYDPGVNFRSVYYPKEKRSIVVCS
NKSGGAFEIISGIEKEII
>lcl|NZ_CP026520.1_prot_WP_042229301.1_1404 [locus_tag=PC41400_RS07325] [protein=response regulator] [protein_id=WP_042229301.1] [location=complement(1618968..1620599)] [gbkey=CDS]
MIALLVDDEKHVRSALRMLIKWEEHGISQIFEAENGEEAIEVIRREKPQIIITDMMMPVKGGVELLQWLQTNDPSAKSIV
ISGHDDFSLVRHTIQYGGTDYILKPIDPDQVNEALAKAVASWNKEEHERSRSVERNIEMNRIKPVYWDRQLSGLLDEPAQ
NPVAVRDLQADWGLPAGTGSCRIAILSMVTMERKLKNKFGRNTDLLFFSLINICNEFLHRRSRGAAFRYWSSNEILLLVW
GEPDKVPALLGDINEGILTALHSRLDFGLSRAHAFPDGLPAAYREARDALRQRNLLSRGGWIHEAVRPEGLQPRRLRFSD
FEEAIRTAIRSGSAEEISSAVQAWFDGVRSLSAITLEQIEQWRREADTMRGRWIEDFFPDGPEELLAEVRSESGSGQTIV
PLNEQGTLSLADWQQELTHSFVQLSKKLLKRQHREKNVIYDIAKYLQTHYHEEISLQDIAGRFFLSREYISRKFKQEFGT
NLSDYLGAIRIDKAKLLMMNPNLRISQVSEMVGYSDEKYFSKVFKKVTGCSPNEYRKANQPIM
>lcl|NZ_CP026520.1_prot_WP_042229299.1_1405 [locus_tag=PC41400_RS07330] [protein=histidine kinase] [protein_id=WP_042229299.1] [location=complement(1620596..1622560)] [gbkey=CDS]
MNNSSIRTRLILLMLAATVIPIVISMIVTYFYTTRSVGTKLIQENANLIYQGKTNVQAYMDTLNRNSLSLYNDPGILLML
RRNVHDYTNEAQTYTLLQNMTHSVDGIYQIYLYMAKTRRSTLLIGDIPKRGYDTDNFSEPYTTGNPSSVVEPTHMSSIYG
LGQSPPYYDPRPVITLHRAIYNVPTTEWLGTLSIDIKLEAIQDIADQLFNKQHERLYLLDQSGMVVYSGDGSLIGKPLSD
AWTSPILESKETRGQLEMNKSVYLYEKMDTPYMKWTLVKEIPHSYLYEGARSLTQINIIIAAVSLLLVIAATLFVTIRIT
APIKKLIGTINQIQAGNLTARVDGEGQDEIGILVRRFRTMMETINNLILREYRLNIANKTNQLKALQAQINPHFMNNALQ
SIGTLALQHNVPRIYSLINSLAKMMRYSMNTNQEAVPLSMEIGHVKAYLELQTERFEGKFRTDFELDEVALGIEVPKMII
QPLIENYFKHGFDPKLGPGHLTIRARVEPGGGTGPGEVPLKPGGEAGPDGAGADSGHAAASARRNAPESRPDEDSAAADS
SPDSGGTSASPVLFVEVEDNGRGIPEDKLTELQAGLAVMREETGMDEEGGIGIGNVRTRLRLYFDPQASLTLENTDPHGV
RITMRIPLGEDTTT
>lcl|NZ_CP026520.1_prot_WP_009676154.1_1406 [locus_tag=PC41400_RS07335] [protein=sugar ABC transporter permease] [protein_id=WP_009676154.1] [location=1622680..1623558] [gbkey=CDS]
MHKRSSQWGQQLFFLGPAVLFFTLIIIIPFLLGMYYSFTNWNGVSGEVSWVGFDNFKRIFTQDQDFWSAFWFTVRFTIVG
VVLSNLVGFVLAYLLTRPLKTRNFLRTIFFMPNVIGGLLLGFIWQFIFVKGFSAIGEMTGIGFFNLPWLGDSTTAFWGIV
IVFVWQTAGYLMVIYIAAFSNVPKDVIEAAQIDGATQGQVLRSIMIPLIMSAVTVCLFLAISWTFKMFDLNLSLTKGGPF
RSTESVAMNIYSEAFQNNRYGLGTAKALVFFIVVAVITWLQVRFTKSKEVEA
>lcl|NZ_CP026520.1_prot_WP_042229298.1_1407 [locus_tag=PC41400_RS07340] [protein=carbohydrate ABC transporter permease] [protein_id=WP_042229298.1] [location=1623559..1624395] [gbkey=CDS]
MEAQNRYSGRVFIAEIVTFVIALLFLVPFYFLLVNSVKSFGDLLSNAASWPESFHFENYKRAWSITKFPVVFWNSLIVTV
ISNIGIALISAMAAYRLVRYKSRINQVLFGMFVAAMVIPFQSIMIPLVKVLSGVGLMDSLWGLIICYLGFGAPMAIFLFH
GFVKSIPLEIEESAVVDGTSPYGVFWRIVLPLLKPMFVTVIILNALWTWNDYLLPSLVLQNQSLRTIPLATFSFFGQYMK
QWDLALPALVLGITPIIIFFLAMQRYIIEGITAGSVKG
>lcl|NZ_CP026520.1_prot_WP_042229297.1_1408 [locus_tag=PC41400_RS07345] [protein=extracellular solute-binding protein] [protein_id=WP_042229297.1] [location=1624534..1625835] [gbkey=CDS]
MKKLSTLALVSVFAMSTVLAGCGSGDSGKTAEGGNAAEGGTKTIKIFQFKVEIAEALNKVKAEYEKTHPGVKLDIQTVGG
GADYGAALKAKFAGGDTPDIFNNGGFQEMNTWMEHLEDLSDQPWVKDLVPVAKEPMTKDGKIYGQPMGLEGYGFIYNKDL
FKKAGIAELPKTLTQLDEASKKLQAAGITPFANGYQEWWVLGNHTLNVAFANQPDSNKFIAGLNAGTEKFKGNPVFEQWT
RMVDLTLKYSNKNPLTTDYNTQVTMFANGQAAMMQQGNWTQVQIDGINKDLNIGILPMPISDDAAQTDKLYVGVPNNWVI
NKNSKVKAEAKEFLNWLVSSDAGKAFITHDFKFIPAFTNIEAKPEDLGDLAADIMTYSKENKVLGWNSSKYPEGAMQEFG
ATMQAYVAGKKTKDQMLDDFQKTWDNLKGKTAQ
>lcl|NZ_CP026520.1_prot_WP_042229295.1_1409 [locus_tag=PC41400_RS07350] [protein=MFS transporter] [protein_id=WP_042229295.1] [location=complement(1626165..1627619)] [gbkey=CDS]
MVSPCKIGSPAGLTDGASGPETASDVPVVEASAPIPAEAEALVQDQTSASVLKPSPVTASASIPAPASTGALSADSQPRS
VSPSAPASLPVPPSAPSSLSASKALLFSAACGLAVANIYYAQPLLDALASEFGVAPAAVGLVVTVTQICYALGLLLLVPL
GDLLNRRRLIVVQMLLSAGALLIVGTAPSAPVLLAGLAAVGLLAVVTQVLVAYASTLASSAERGRVVGVVQSGIVIGILL
ARTFAGTLTDLAGWRSVYLISALLMFVTGAALYRILPDVHHTKTELSYPRLLRSVFALFIEERLLRIRAVLAFLIFTVFS
TLWTALVLPLSAPPLSLSHTAVGAFGLAGAAGALAAARAGRLADRGLGQRTTGLALVLLLASWGFIGFTPHSLAAAVIGV
ILLDLAVQAVHVTNQSLIFTVRPEARSRLTAGYMIFYSLGSATGAIASTSLYAIGGWNGVCLFGAAVSGLALLFWAVTRG
NGPE
>lcl|NZ_CP026520.1_prot_WP_042229293.1_1410 [locus_tag=PC41400_RS07355] [protein=TetR/AcrR family transcriptional regulator] [protein_id=WP_042229293.1] [location=1627876..1628457] [gbkey=CDS]
MVRQREFDEDAALDAAMRLFWEKGYEAASLTELTARMGIQRPSMYAAFGSKKELFEAALRSYTKAHAARVRAKLQSRTSV
KDAFRSLFEGIVADEYEGSLNRGCFSINTMVELAPHDEKFEILTREHQLYLSVIFQETIERGIRTGELDAALEARGTAQA
LLVSLIGLTVMLKSRPDRQFADNTVKTVLTLLR
>lcl|NZ_CP026520.1_prot_WP_042229292.1_1411 [locus_tag=PC41400_RS07360] [protein=NAD(P)/FAD-dependent oxidoreductase] [protein_id=WP_042229292.1] [location=1628701..1629696] [gbkey=CDS]
MTYDSIIVGGGIAGLQAAIQLGRYDHRVLVIDAGYGRSTLCANYHNVLGYPDGISGEELRRTGRGQAEALGVAFAEGLAV
RLAGSLETGFRVGVRKLLPDERGGSHPLAAHEPGGRGLPDESGAGPETEYTGATLLLATGVMDRFPELPGLRECLGHTVY
VCPDCDGHEVAGRRTLVLGAGNPGAGMALTLRPRTDKLTFVNHERRPVNADLQEKLERAGIDTVAGPIAAILEDDGRIGG
VKLAGGETLEAERGFLGFGGNAVHSGLASQLGVERMENGHLLADPRSKETNIPGVYVAGDIGVHAEQLTVAMGEGAMAAI
WMNKAIRKKNN
>lcl|NZ_CP026520.1_prot_WP_042229290.1_1412 [locus_tag=PC41400_RS07365] [protein=hypothetical protein] [protein_id=WP_042229290.1] [location=1630015..1630659] [gbkey=CDS]
MAEYAKILRWTKSDAVYGLKTNIQLPSSASVGTTGGFINFYLGVYGSNTNYECGLSTNYTEALNGKWHWFWNTGAANDGG
AFNLAAGSTVPIELSVNSSNKLDFKVNGTVVKTFSGATGTFGSLTSARLVVAACDRSFTTVPNPLPNWNTLHNQVVCSGI
SFRNSAGAWTVLSSANSTPPSSTTFWPDGYSHTGTPADYTKVVSTGQLIASLKR
>lcl|NZ_CP026520.1_prot_WP_164977111.1_1413 [locus_tag=PC41400_RS29670] [protein=hypothetical protein] [protein_id=WP_164977111.1] [location=1630861..1631013] [gbkey=CDS]
MNERSNLKREKKPWSLTIRLLNILLILVILGIALYVGLWIVGIYGMNAKG
>lcl|NZ_CP026520.1_prot_WP_164977112.1_1414 [locus_tag=PC41400_RS29675] [protein=hypothetical protein] [protein_id=WP_164977112.1] [location=1631926..1632072] [gbkey=CDS]
MNERSNLKREKNPWSLTIRLLNILLILVILGIALYVGLWLVTILGINI
>lcl|NZ_CP026520.1_prot_WP_042229284.1_1415 [locus_tag=PC41400_RS07375] [protein=WG repeat-containing protein] [protein_id=WP_042229284.1] [location=1632233..1633750] [gbkey=CDS]
MRKLVGFILVFTLVFSMMAVTNANEASGSGSVPQPDYLPYVEVGVLSEGLAFYMEAGEETDSFGFIDTSGKIVIKARFEA
AGIFHDGLAPVRIDGLWGYVDKTGQFAVPPIYTGVGRVFEEGFAQVVKDDKWGYVDKTGKEVIAFDYEDTNPFNNGLAPV
KKNEKYGYINKEGKEVVPFRYEAATVFSEGRAMVKIKDKWGVIDQNGIEIEKPKFEEAFPYSRGLAAVKHEGKWGYIDLS
GREVIQAHYDEARPFQEGDLAPVQVDGKWGYVNKEDTMIIRPQFMDVYTVFSEGLALVSIENEDDYEMLGYIDQRGKVVV
PYSYDLMGSSFSGGLATVADGGQGSFYFMRNPLNVQTQEPQPQSALPSASKVMINGREVSPEAYNIGGNNFFKLRDLAML
LNGTNKGFEVGWDQDAKVIRLTTSKAYTEVGGELAATGQLSVKQAAPAAAKLLVNSEEARFTAYTLDGNNYFKLRDLAQA
LDFNVSWDAASNALTIDTKSGYQPE
>lcl|NZ_CP026520.1_prot_WP_042229282.1_1416 [locus_tag=PC41400_RS07380] [protein=DUF2161 family putative PD-(D/E)XK-type phosphodiesterase] [protein_id=WP_042229282.1] [location=complement(1633786..1634895)] [gbkey=CDS]
MAIRSETELYGPVKAYWEKLGYDVRGEVRHCDLVAVLEGEPPVVVELKRSLTVPLLLQGLRRQDVTHRVYIAIELPAKGR
VPHRMTWADIRKLCGRLGLGLITVQFYKTRKPAVEVVCEPPGFGDTAPARKTRKRAAERVLGEFRERSADYNVGGSRGLK
LVTAYREKALHCALLIREQGPLSPRRLRELTGNAKTPALLQRNVYGWFVRVRRGLYGLTRPGEEALDTFAHVAARFMPRE
PDASPDAGLTESGEHSRPEAGAADERGGAAGAGPTTRVRLPEPPETVHPRPEAGVAAMAELPGASEPADVRDAQDVSEPA
DSPDLRNASSQRDITAQTDKRDKPDASGLQQTPSTRKRKSAKAKKGGST
>lcl|NZ_CP026520.1_prot_WP_042229280.1_1417 [locus_tag=PC41400_RS07385] [protein=PrkA family serine protein kinase] [protein_id=WP_042229280.1] [location=1635150..1637045] [gbkey=CDS]
MDIFKRITEHRTEKEQLAWSGTLREYLELVRQNPGLAMTAHARVYEMIASHGIEQNDGHTRYNFFDKEIFGLDRAVEKLV
EEYFHSAARRLDVRKRILLLMGPVSGGKSTIVTLLKRGLEQFSKTDKGAVYAIKGCPMHEDPFHLIPNELRPEIESELGI
RIEGNLCPSCQLRLKTEYGGNIENVLVERVLISEDDRVGIGTFSPSDPKSQDIADLTGSIDFSTITEYGSESDPRAYRFD
GELNKANRGLMEFQEMLKCDEKFLWNLLSLTQEGNFKAGRFALISADELIVAHTNESEYKAFIANKKNEALQSRMIVMPI
PYNLKVTDEEKIYTKLIKQSDMSHIHIAPHALRAAAIFSILTRLKETKKQGMDLVKKMRMYDGEAVEGYKDADLKEMQNE
YLEEGMSGIDPRYVINRISSALIRHDLQCINGLDVLRALKEGLDQHPSITRDERERYLNFISVARKEYDDIAKKEVQKAF
VYSFDESAKTLFDNYLDNIEAYSNWGKIKDPLTGEEMDPDERLMRSIEEQIGISENAKKAFREEIQIRISSYSRKGKKFD
YSTHERLREAIEKKLFADLKDIVKITTSTKTPDERQLKKINEVTRQLIDGHGYCPVCANELLRYVGSLLNR
>lcl|NZ_CP026520.1_prot_WP_042229279.1_1418 [locus_tag=PC41400_RS07390] [protein=MATE family efflux transporter] [protein_id=WP_042229279.1] [location=1637252..1638640] [gbkey=CDS]
MAKNEAQMDLTQGPIARKLLLFSLPVLYGNVLQSLNGTVNSIWVGKFLGESAFAATSNANNVMFLMLSSIFGIGMASTIM
IGQSIGAGDTAQAKRVVGASALFFFTLSLVVGILGVLLSPQILQWMHTPADAMDMANAYLRIIFMSMPFMFCYNMIMTIL
RGAGDSKTPFRFLVLSIILDIALNPLLIFGWGPIPALGISGSALATGISQLVALVAILVSLYRKKYFLRITREDRHLLRF
DGAIIGSLVKKGVPMGLQMIVVSTSSLAMINLVNRFGSEATAAYGAAIQLSSYIQMPAMAIGGAVSSFAAQNVGAGRWDR
VHRVTGVGVMYNVLMTGVLVCLIYLFNRSALSLFLPEGQAIEIGMRINLLTLWSFIIFGITFVVSGVVRSTGAVMFPLLI
TFLALWCVRIPLSYYLADSYGLDAVWWSFPVGFTLGSLLSVAYYFWGNWKKASMLTGGVQKA
>lcl|NZ_CP026520.1_prot_WP_042229932.1_1419 [locus_tag=PC41400_RS07395] [protein=hypothetical protein] [protein_id=WP_042229932.1] [location=1638816..1639223] [gbkey=CDS]
MTFETQGQGTYGQDIYGQGAYSQDIYGQGTYGQGTYGLQQEWNQAEWVSTEADEEARKQGEYAVPGGYPGGQGGYDGGQV
GYHDGGYHGGYHGGYHGGYHGGYPYYPYHHHYYPQPYCPYYPPYYNYPHYYPYNR
>lcl|NZ_CP026520.1_prot_WP_042229276.1_1420 [locus_tag=PC41400_RS07400] [protein=hypothetical protein] [protein_id=WP_042229276.1] [location=1639394..1639660] [gbkey=CDS]
MLRSISDTLFLLALILLIVSAIVSGIGFQAFTMRRKDNDPRQIEVPNKQKQLEKQNKGMAGFIRSKITLTALVFALVSVV
LSYLERYV
>lcl|NZ_CP026520.1_prot_WP_042229273.1_1421 [locus_tag=PC41400_RS07405] [protein=class I SAM-dependent methyltransferase] [protein_id=WP_042229273.1] [location=1639822..1640598] [gbkey=CDS]
MNEAEQGKWDAGHYDTQMYFVSRLGQGVMDLLSPVPGESVLDLGCGTGTLAHEIALLGAQVTGVDFSEAMIEQARQNYPE
LAFEVGDARTYRSGRTFDAVFSNAALHWVPQADEAARTVAEALRPGGRFAAEFGGFGNVARIERAIHEAVREAGIDPEPR
NPWYFPTIGEYTGVLERAGLDVRYAEIFDRPTPLPGGEQAIRHWLHAFAGSFFAGIDAATQERITQGVEERLRPALFRDG
AWTGDYCRIRVLARKPLG
>lcl|NZ_CP026520.1_prot_WP_042229272.1_1422 [locus_tag=PC41400_RS07410] [protein=LacI family DNA-binding transcriptional regulator] [protein_id=WP_042229272.1] [location=complement(1640885..1641952)] [gbkey=CDS]
MSKHVSMQNIADRLGISKYTVSQALSGKDGVSDETRLKIMETARTMGYRKKINKSPAAAGLFQERDGRRKEITSSSSDPY
MLVWIHSAKQQDPFFWPKVIDGLAQTCRDLQADYMLISVPPHREDELAMPGFLDPARCMGHILLGTFSTRAVISLQQTGL
PLVLVDHEEPLVDVDCVVNSNVDAAVAASKQLFREGCDSLVFVGSDSFSVSFRERALGCRIAAEEAAESGRPVCLSKWNV
PYASPTWPHDLAVRVGSVSTSELPDGFICANDHIALRLMALLKQRGFDIPGQCRVIGFDNIEAAAASSPTLSTVELSKEL
LGIRAVETLLYRKRHPGRAHEKIVLAPEFIARASG
>lcl|NZ_CP026520.1_prot_WP_042229271.1_1423 [locus_tag=PC41400_RS07415] [protein=DUF817 domain-containing protein] [protein_id=WP_042229271.1] [location=1642166..1642981] [gbkey=CDS]
MIRSVRYGIRQLLLFGWLEALSCLFPVLIFISLAVLKSANVPHMYDLMLVICLLIQWAMVRTGLETRDEVKVICLFHLIG
LALEIYKVNMGSWSYPEEAWTKVWGVPLYSGFMYASVASYLCQAWRRLHVELTRWPSPWFTVPLGAGIYLNFMTHHYIWD
LRWVLTALLFVLFFRTAVLFTVGRQTYRMPLVLSFFLIGFFIWLAENIATFLGAWQYPNQRESWQLVSFGKMSSWFLLVI
VSFLIVAQLKRVKQSENKPDGRMVPAAVPEE
>lcl|NZ_CP026520.1_prot_WP_042229269.1_1424 [locus_tag=PC41400_RS07420] [protein=MATE family efflux transporter] [protein_id=WP_042229269.1] [location=1642985..1644328] [gbkey=CDS]
MFTNWGMILRLAVPSIVSFAAATLTGTINLMMVGPMGALVIAAVGICNIIMYNAWAACSGIGHTVNYLVAQNFGAGDMKK
GVERTYIALYLCLALIAVILAAGWLGSEGILRLMGSSEELVAAGKEYLELRFYAMAFSVLNFALFGFLRGIGDTRSPMAL
TILGNAAMIVFTYGLTYGHLGFPELGLPGAGLAVLIGEAICSIGSVYIVFFRLHDKYRTRTRVKLDRREARLIAGESGKL
GMQEFAMSLAMFVFTMFVTRLGTEALAANEIALNVMSLGFMPAFAFASTATILVGREVGKHRPDLARRYGTDTAVLGSIC
LLVLGAVEFVFAVPIAKLYSQDPEVYVLAADLIRISAFLQLFDGLLNFYAGGLRGIGDTTFLLGTSLVLSWVFFVPLAYL
LTFVWGFGSAGAWISLYSFIAAFGLAVMYRFYRTDWAAVRIKEAEGR
>lcl|NZ_CP026520.1_prot_WP_053228698.1_1425 [locus_tag=PC41400_RS07425] [protein=lysoplasmalogenase family protein] [protein_id=WP_053228698.1] [location=1644590..1645285] [gbkey=CDS]
MMGWLWAVQVILFAAGTGLGMSDARRKSTEVRLPLWVRMLISLTLTAAAFALWRDDPASSSKEWIFWGMAWSFIGDLSMA
GIIPWRHPMIGGMLTFAVAHVFYIAAFAAALHHEGLDLMNAGLAVVAVFYGLVLIAAWLLLIRNPDAPRLFNIGSLVYGL
WIGAMASAAAAMAWSLGGALWLTAVGGLLFVVSDTIIGAAAIGGRLLRHQGYWVWLTYVAAQMCIVYAFTL
>lcl|NZ_CP026520.1_prot_WP_042229268.1_1426 [locus_tag=PC41400_RS30065] [protein=hypothetical protein] [protein_id=WP_042229268.1] [location=complement(1645447..1645635)] [gbkey=CDS]
MDTLTDDVLNLIDQVDTDPIMQADESIYLPFTIQYVVICPIILLPEIPRSVQDRTGRIPACH
>lcl|NZ_CP026520.1_prot_WP_241688811.1_1427 [locus_tag=PC41400_RS30195] [protein=hypothetical protein] [protein_id=WP_241688811.1] [location=complement(1645875..1646264)] [gbkey=CDS]
MSIYRFDPGTGTDTVPGNAAVIFNGVFANSGGAFTFVPPSATITINETGTYLISFTVHNQGNADFNLTVSSAPVTPVPFT
GNGGGPTSAEVIIVVTAVPKTIQLVNANATPAQLHNLTNTTLTILKLTP
>lcl|NZ_CP026520.1_prot_WP_042229267.1_1428 [locus_tag=PC41400_RS07445] [protein=hypothetical protein] [protein_id=WP_042229267.1] [location=complement(1647241..1648278)] [gbkey=CDS]
MSFFKWFFTSMKSQPVQKQPKAPPVQLFSSFAPYGLGVLENDSLKPVEARFNEVFGKYAKYWDHVKARVLKENPDISETE
YSWIFLEFKRFLALNIYLNNVGMYSTSVDDIWHEALMFTRDYQEFCRDLLGEMIHHSPHVGDAPQPGEHERALFELVYSL
VFRFYPENEQLLGKFGQRRLHPEFVAYARDFDWNRIRRDYFKPNTIVIYEDAVQDLVHHLFNGIQNAPPAAAARNSGRMS
SAGRTAASSFNANDVSGAGYGALSALLFYSILNGNDDREGGGGNGSGDSDCSSSSCSSSDSGGSSDNNSSGSGSSCSSSS
CSGSSDSGSSCSSSSSCSSSSCSSS
>lcl|NZ_CP026520.1_prot_WP_042229265.1_1429 [locus_tag=PC41400_RS07450] [protein=MerR family transcriptional regulator] [protein_id=WP_042229265.1] [location=complement(1648376..1649167)] [gbkey=CDS]
MYSVGELARKANITVRTLHHYDQIGLLKPSGSTEGGHRLYSDEDVMRLEQISILKKMGLGLKAVHEMLDSYGTDWLSCLE
DQLQMVQREKEKLEELESMLRVTIHSVQIEGTLNWNAMFQMIRLSQKDPAAKQRFIREHFTAEERKLLHKLPNLNKDDPL
TMRWTEEVAQAKAVMHEDPASPAAQEVAARITTLALEAFDGNHTLMNKYWELHGAEDNPMNLYPFEPELLEFLNAAFEYY
DRTQTAHEAAAPPKPFVGKNRRK
>lcl|NZ_CP026520.1_prot_WP_241688812.1_1430 [locus_tag=PC41400_RS30200] [protein=DUF6254 family protein] [protein_id=WP_241688812.1] [location=1649467..1649592] [gbkey=CDS]
MTQSKNRKEGLAKSRKQRQNPHGKIRSLKEIAAETHQGIEE
>lcl|NZ_CP026520.1_prot_WP_042229264.1_1431 [locus_tag=PC41400_RS07455] [protein=cation diffusion facilitator family transporter] [protein_id=WP_042229264.1] [location=complement(1649899..1650891)] [gbkey=CDS]
MNDSKKRKAAGAASGQIHRHDHGHSHDHSHDHEHGGIGHHHGPSSKKGLFIAFLITTVIMVLEFAGGLLTNSLALLSDSG
HMLSDAASLLLSLVAMWFAARPASPRRTYGFYRVEILTAMLNGITLVVLSILIIREAVERFAHPPAIASGTMTLIAVIGL
VANLVSAWFLTRMGGAKDNLNVRSAYLHVLGDALGSVGAIIAGLLMQAFQWYQADPIISILVSLLILRSAWGVLKSATHI
LLEGVPSRLDADDVRSALLRIPGVREVHDLHIWTITSGMDSFSCHLLIDDEADSQEVLQQAVACMENEYKIRHATIQVET
RHLKHAEFSI
>lcl|NZ_CP026520.1_prot_WP_042229263.1_1432 [locus_tag=PC41400_RS30555] [protein=metal-sensitive transcriptional regulator] [protein_id=WP_042229263.1] [location=1651191..1651586] [gbkey=CDS]
MVLHSEKDNGRTEGETGSAPLASADAGTSCHTHTRDEDTERRSHHSDKVKNNLVSRLNRIEGQIRGVKGMIEKDTYCDDV
LNQIAAVQSALNSVGKLLLEGHMKSCVIERIQEGDTDVIDELLVTVNKLMK
>lcl|NZ_CP026520.1_prot_WP_042229261.1_1433 [locus_tag=PC41400_RS07465] [protein=copper ion binding protein] [protein_id=WP_042229261.1] [location=1651620..1651820] [gbkey=CDS]
MEKLTLKVEGMSCQHCVNSVEKAVGELGAKAQVHLAEGRVEVEYEAGAVKPEQIKEAIEEQGYDVV
>lcl|NZ_CP026520.1_prot_WP_042229259.1_1434 [locus_tag=PC41400_RS07470] [protein=IclR family transcriptional regulator] [protein_id=WP_042229259.1] [location=complement(1651946..1652698)] [gbkey=CDS]
MEHHLSSVKNSSRLLKEFLDGPKELGVTELSRRLQLSKGAVHKLLATLESDGFISQNPSTKQYSLGFTLLELGTKVLRDH
NLVDFAKPYLQQLAEKTKELACLCVVDGKDAIYVDKIDSQHPIRFNVDAFRRFPLYATSASRVMLAFRPEAFIDSVLADG
IRTYTPHSMQAPDEIKERLAAIRSRDYEISTNLRNVGVTGMAAPIRDATGEIVASVSLIGPSDRVVPQQDAFLQQVLLTT
RNISFGLGYR
>lcl|NZ_CP026520.1_prot_WP_042229258.1_1435 [locus_tag=PC41400_RS07475] [protein=ABC transporter substrate-binding protein] [protein_id=WP_042229258.1] [location=1652961..1654070] [gbkey=CDS]
MKKKIGVLVSVAALCGSLLAGCGSSKEPAKDAAASPGASTSALQPEKELVVAGNGATVEELMKDEIFKRFKAKYPDIKLT
YVSGVSTEIVAKAKAVKNDPQTDVAIIEGGEQEIGRKEGLFEPLKEADIPNMKNVIADLKVAEDSGVAVNFTPMGISYNE
AIVKEKNLPIPESWNDLARPEMKGNLTLTEISSNFGRSALIMLAYANGGSEKNMDPGFEKLKTIAGYMPTFAKSAAQLQQ
NLQNKSAAYTSWTMARSIVQKDQGIELKFVVPKEGANLVPNVATMTKNAKHPNAAKLFIDFLLTDEVQTLYASKLYYNPA
TSVKLPEDISKKIEFDRSKIVKFDYETVGGSTAGWIDRFNKEIAPSVGK
>lcl|NZ_CP026520.1_prot_WP_042229257.1_1436 [locus_tag=PC41400_RS07480] [protein=ABC transporter ATP-binding protein] [protein_id=WP_042229257.1] [location=1654093..1655187] [gbkey=CDS]
MEKKIDVELNAVRKTFGSNVVVDNFDLKIEQGECISFLGPSGCGKTTTLNMIAGFLDPDGGEISIKGKRMNGVPSNKRDL
GMVFQTYSLFPHMTVAENVAYGLKLRKMPKAEIKDKVRKILELVKLPHVADRYPKQLSGGQRQRIAIARALVIEPSLLLL
DEPLSNLDAKLREELRDELKRLHKETGVTTIFVTHDQEEALYLSDRIVVLDHGKVEQIGTPWDIYNRPASEFVHTFIGKS
NRLPGTVRGSANGGMAVQTDAGFVLEAFADGRTFTTGESVVAYVRPEKIRLTESGALADTVRAAGTIRQLSFLGSFVECE
VEVGSALVTVRMQMQDGLADYTAGRQVGLTWNPKDVFVFPRKEQ
>lcl|NZ_CP026520.1_prot_WP_042229256.1_1437 [locus_tag=PC41400_RS07485] [protein=ABC transporter permease] [protein_id=WP_042229256.1] [location=1655191..1656057] [gbkey=CDS]
MQAKSSRFRRLSSWTGLFLLAPGLIVLLGAFLIPMIYILLQSLQDAEQQFSLHNYLLFVQDPYYAGILWRTIKLSLYTVL
LTLILGYPVALFMSKASGKMRGLVTFLIISPHLISVVIRNFGWVVILGDKGWVNSSLIKLGLIDKPLKLLYNELGVVIGL
TDSFIVYMVLALATSLYAVDQSLYKAAGILGASKARSFFSITLPLSLPGIFAGTTLVFSLSMSAFVTPALMGGSSVKVMP
VITYEKIMATLDWPLGSALAFLMLASTYALVTVFTKIVETKRYREVFS
>lcl|NZ_CP026520.1_prot_WP_042229928.1_1438 [locus_tag=PC41400_RS07490] [protein=ABC transporter permease] [protein_id=WP_042229928.1] [location=1656057..1656857] [gbkey=CDS]
MRRIPWLGIVTFIVLLFVISPLLVIIPTSLTSANYLSFPPEGFSLKWYEKILDRPEFIESFIFSLELAALSAVLSTIIGT
LAAMAIHKYKFPGSGFVNALLLSPLTVPSIIIGMAALLFFTRIGLAGTFTGLLITHMLISIPYVVRLTMTGLSAYDYTLE
RAAYILGASPFQVFRDITLPLLRPAILSGMIFSFLTSFDNVTVSLFLVSPETTTLPMAIFSYMQETLNPLVASVSSVVIL
LSLVFIYILERVYGLDKLFGTNSHSH
>lcl|NZ_CP026520.1_prot_WP_042229254.1_1439 [locus_tag=PC41400_RS07495] [protein=M20 family metallopeptidase] [protein_id=WP_042229254.1] [location=1656996..1658153] [gbkey=CDS]
MSDVTLVENYMPELLRLLEESVNMDSPSRHKALCDRMADWYEIQFRRLTGGTVTRLPHEVSGDRLLCEWGTGSRRILLLG
HYDTVWPEGEASRRPYGVRDGRALGPGVYDMKLGLLQAMFALKLLGDEGRGTAEDVTVVLLINSDEEISSPSSRGLIEQT
ARGCEAVFVLEPPMEPEGALKTGRKGSGRYKLLVEGIAAHAGVDPSKGVSAIEELAHQIQHLHGLTDYGIGTTVNVGIVH
GGIGTNVVADRAEAEIDVRVKTSAEAERVHALIAGIRPHLPGARLRIEGGMMRPPMERTDAIARLYGIARRIAADELGFE
LGETATGGVSDGNFTAACGVPTLDGLGARGDFAHSPDEYVRLDEIAPRTALLARLIETVAAGERR
>lcl|NZ_CP026520.1_prot_WP_241688813.1_1440 [locus_tag=PC41400_RS07500] [protein=hypothetical protein] [protein_id=WP_241688813.1] [location=1658413..1658814] [gbkey=CDS]
MLDVGVICEELGLRKDYIENVVKKNGGMKPGAREAEDEQGDGEKDEEDEDKKDADKQKEKESGKEEWLKLGLEEGREQGL
EQGLEQGLEEGWEQGMEQRQEHGQRDFSALSDNEGLPSRSRFRFSFFMEGVGS
>lcl|NZ_CP026520.1_prot_WP_042229253.1_1441 [locus_tag=PC41400_RS07505] [protein=YafY family protein] [protein_id=WP_042229253.1] [location=1658882..1659832] [gbkey=CDS]
MMDKVIRLFHIILAIQANPGISASDIARKCGINPRTVYRDLATLSLIAPITNEGRGTGYRFMGKFFLYPLNFTEQEALVF
SMLPSVLNTEELPPGFDQAYHKVMATHGKEKSHHRDIVQNITDIIQMGAPAYRQEHPNFLEPIIRAVLDQKTISTVYHTM
SRNKTAERNIDPYYLVPREQRFYLIGYCHLKREVRTFRMSRFQQVEVTGQTFEKGDFNIRQYLKNTWSIERGNKNMTFKV
KFNAEIARYVKEEELFVHPRMRDRKDGSLIFEVTVNNEREFLNWVMQYGPAAEILEPKAVREKLKQQLHQWVELYK
>lcl|NZ_CP026520.1_prot_WP_241688814.1_1442 [locus_tag=PC41400_RS07510] [protein=hypothetical protein] [protein_id=WP_241688814.1] [location=complement(1660197..1660649)] [gbkey=CDS]
MTNELMPQYAYAGEPYWGLSTVTADRQPHPVLVEARLTAPNQILMTYDQRSDLASATNVSNYWIRSNMGPAGIASVGMNE
ALTAANAIRPDMAMITPADNSRMRFLMTFTVNAMRGVMYTVLPCFVNLEGMTSFRGENWGPLSRNMFIGM
>lcl|NZ_CP026520.1_prot_WP_042229252.1_1443 [locus_tag=PC41400_RS07515] [protein=glycine C-acetyltransferase] [protein_id=WP_042229252.1] [location=complement(1661048..1662226)] [gbkey=CDS]
MSGLDYLAEELEQLKREGRYRLPAVWESGSDVWMELNGRRVLQLSSNNYLGFANHPDLKRAAADAVYRYGAGSGSVRTIA
GTLKIHDELERELAAFKGTEAALVFQSGFTTNLGVFSTLLQDGDVVISDELNHASIIDGIRLSKAARKIYRHKDMNALED
VLRASGQYRKRFIVTDGVFSMDGDIAPLPAIVELAERYDAIVCVDDAHASGVLGRNGKGSTDHFNLHGRVHIQIGTLSKA
IGVVGGYAAGSQALKEVLIHKARPFLFSTSHPPSVAASCLEAIRVLRSSSELVERMWSHAHGFRKALIQAGFDTGASETP
IIPIIIGDAAATMHFSSRLLEEGVFAQGIVFPTVAADKGRIRLIITAAHSQDDLDFALEALKKVGRETGLIS
>lcl|NZ_CP026520.1_prot_WP_042229251.1_1444 [gene=tdh] [locus_tag=PC41400_RS07520] [protein=L-threonine 3-dehydrogenase] [protein_id=WP_042229251.1] [location=complement(1662245..1663291)] [gbkey=CDS]
MTQTMVGLVKAHRRAGAVLMEVPVPECGPDEVLIRVKASSICGTDMHIYQWDDWAEQHVVTPNVFGHEFAGVVHEVGSQV
SRVRVGDYVSAEGHIVCGMCKQCRSGNAHVCPHTRSFGITAPGCFADYAVTKAVNIIPNDKSLPFELACLQDPLGNAVQS
VLSGDIVGKSVVVMGVGPIGLMAVQVAKAAGAGKIIAVDLNAYRLDMARKLGADATILSSPAVSAPDLIREMTGGEGAEV
GLEMSGNARAINDLLEAMSPGGRVSLLGIPAHSVPIDLSRHVIFKGLQVHGITGRRMYNTWHQIKGLLDQNRIDLQPLVQ
HTFTLDRYEEAFDLMASGQCAKVVFTHI
>lcl|NZ_CP026520.1_prot_WP_009676228.1_1445 [locus_tag=PC41400_RS07525] [protein=XRE family transcriptional regulator] [protein_id=WP_009676228.1] [location=complement(1663523..1664077)] [gbkey=CDS]
MDNMVERVAGNLKKIRRSRGLSLDKLAELTGVSKTMLAQIERAESNPTITILWKIAHGLHISVSALIEDEKPSVTLVKKS
ELFPMSSEEGKYTAYPLFPYDGSSRFEIYSVELQPGCEYESDPHHEGVEEYVSVTRGTLEMQIGEQVIVAENGDAIRFHA
DQSHIYRNSTDEVVHIQSVILYPF
>lcl|NZ_CP026520.1_prot_WP_042229249.1_1446 [locus_tag=PC41400_RS07530] [protein=LacI family DNA-binding transcriptional regulator] [protein_id=WP_042229249.1] [location=1664262..1665290] [gbkey=CDS]
MTRLKDIADVIGVSISTVSRAISNDTSRPVSEETKRKIREAALKLGYPLDDTPAAAASRTQIACVIPGKLLGNHPYFSEV
LQGFHGKMAEMGQPAAIVRTFEEVNDAERFQAMLTESGARGVLALGWYDKELHELLDKQGIPILGVSLNDDSLKMPIVDC
DRVSAARMAVGHLLDQGHTKIGYVGGPAFFRKLENDERFIGFQFAMLQGNLPIRREWVVNTNWDVDLSYRLVTGMLQELP
RADWPTAMFCASDMLAIPAMRAALEQQCRIPEDIAFVGMDNIGFSQYTSPPLTSIDVPRQEIGKLAAKFLCDYVEEDYRI
PPRILLPCSLIVRESSVHDIRP
>lcl|NZ_CP026520.1_prot_WP_241688884.1_1447 [locus_tag=PC41400_RS07535] [protein=ABC transporter permease subunit] [protein_id=WP_241688884.1] [location=1665714..1666616] [gbkey=CDS]
MTLGRRILRNWQLYLLISPVVLYYIVFHYVPMYGLQIAFKDFVANKGIWGSAWVGLDHFQRFFNSYFFQRLITNTIGLGL
YTLALGFPIPIILALLMNEVQSQKFKKFVQTVTYAPHFLSTVVVVGMMMMFLNPRYGLVNTFLGMFGVKPVNFLADPSWF
KTLYVLSDVWQTMGWSSIIYLAALAGVDNQLHEAARVDGATRLQRIWHINLPGIRPTIVILLILNLGSVMSIGFEKVLLM
QNTLNMQSSDIIDTYVYRSGILDADYSFSAAVGLFSTVINFILLISVNWISKRFSGSSLW
>lcl|NZ_CP026520.1_prot_WP_042229245.1_1448 [locus_tag=PC41400_RS07540] [protein=carbohydrate ABC transporter permease] [protein_id=WP_042229245.1] [location=1666632..1667513] [gbkey=CDS]
MENTREDRLFNTVIYLVLGLVLLIVLYPLVFVLSASFSEPQYIMNGEVLLWPKGLTLDSYTKIFEDGQIMRGFGNTLLYT
TTGTLLNVIMTILAAYPLSRKDFVGRNLFMGLFVFTMFFSGGLIPTYILIKNLGLLNSFWVMILPGAVSIWNIIIMRTFF
QQTIPGELEEAATIDGCTNMQILTRIVLPLSMPIIAVTILFYAVGHWNSFFNALLYLSDKSKFPLQLVLREILIQGQTDS
LVKMSTESAIKQQRAVEGIKYAVLVLANLPMFLLYPFLQRYFVKGIMIGAIKG
>lcl|NZ_CP026520.1_prot_WP_042229243.1_1449 [locus_tag=PC41400_RS07545] [protein=extracellular solute-binding protein] [protein_id=WP_042229243.1] [location=1667571..1669187] [gbkey=CDS]
MKRKRAMKLVLACALTTAVLSGCSSGGGKGAGEGAEAKGVNATGFPIVQEPVTVNAFAAKFFANADWNKLKLWQEYEKKT
NIHINWETVQKDALKEKRNLLMASSGYPDLFYASAFSKSDLFKYGKQGVFLPLNEYIDQYAPNLKALMEKYPVIKKGITM
PDGKIYSLPTLYDPEFRSVFFNTPWVKTEWLTKLGLKEPTNLDEFAAMLKAFKQGDPNGNGKPDEIAWGGVGTTGIVGYL
RGSFGLGNHGNANQYVDTEPGSGKIRFVPADPRYKEMLEYIHKLYKDGLLEQDIASVKSEEIDAKGTAGLLGIVDNVDPA
AIYNQKGYAGLPVLKGPHGDQLYTYIGSPLGNIGMFILTDKAKNPAAMVRWMDYFYGEEGIKMFFMGWKDDTYKEESDGT
VDYVDAIKNNKDGLSLDQAVSQYLIWPGGYYPGFVKQAYFKGAEGLPTSVENSKKADPHVLKPEEIWPAFNFTTEEQQEL
TTLANDIETYVTEMRDKFIVGDAPMSQWDDYVATLNKMGLDRYKQIYQAALDRYSKSS
>lcl|NZ_CP026520.1_prot_WP_042229241.1_1450 [locus_tag=PC41400_RS07550] [protein=Gfo/Idh/MocA family oxidoreductase] [protein_id=WP_042229241.1] [location=1669202..1670476] [gbkey=CDS]
MSELTAILIGAGARGAGGYAPYALDYPHELQFVAVAETDAGRREKMAEKHGIPAEHRYDTWEELLNLPKMADIAVICTQD
RMHYEPTMRALAAKYHVLLEKPMSPDPRECLDMEQAARDNDRLLTICHVLRYTPFWTAVKTAITEGRIGEVVSIQLNENV
GYWHIAHSFVRGNWNNSHKASPMILAKSCHDMDVLSWLMDKPCNRVSSFGSLMHFNGAHAPEGSADRCLDCAVEPTCPYS
ATRFYLSQEFRGWAGHFTSDLSKESIVKGLRETDYGRCVYRSDNNVVDHQVVNMEFEGGATAMFSMCGFTFEQERRIQIM
GTRGELRAEGDSIIVYDFLTHQKTEITIPAQTAGHGGGDSGIVRSFLQEVRSYSGQESLTSAAASVRSHLMAFAAEESRL
NGGAPIELTEYARTLAQNAGGLSG
>lcl|NZ_CP026520.1_prot_WP_042229239.1_1451 [locus_tag=PC41400_RS07555] [protein=HAD-IA family hydrolase] [protein_id=WP_042229239.1] [location=1670638..1671261] [gbkey=CDS]
MKPILIFDFDGTIADSKGLAVQLFNELAVKFGGRRIEEAEIGKLSGLSVPDRLKALHVPLFKLPALLLEAKRNYKQAAVS
LQPADGMKELLLALKEEGRTMGILSSNTAENIRHFLDLNGLDVFDFVHSATNLFGKDKAIRSMLRTHSLPADRIVYVGDE
LRDVEACRRAGVKAAAVTWGFDSEELLAGAGPDYVCRTAEQLAAVLR
>lcl|NZ_CP026520.1_prot_WP_042229237.1_1452 [locus_tag=PC41400_RS07560] [protein=hypothetical protein] [protein_id=WP_042229237.1] [location=complement(1671216..1671533)] [gbkey=CDS]
MNGKRKRLICQFRQIRRFYSGVRIESRIEPDLHPLPCGHRLRSRCAASYVPAATTSADATEASTRADIAAAAPGPGLCPE
MLSRFSRYAPPYRKTAASCSAVRHT
>lcl|NZ_CP026520.1_prot_WP_042229236.1_1453 [locus_tag=PC41400_RS07565] [protein=alpha-L-fucosidase] [protein_id=WP_042229236.1] [location=complement(1671559..1672848)] [gbkey=CDS]
MVQQWFEDAKLGIFIHYGIYAVNGIAESWSFYNGRISYEDYMKQLGGFTARNFDADKWADLIEKSGAKYAVLTTKHHDGV
ALWDTRTSDLNVVKRTPAGRDIIREYADAVTRRGIRLGLYFSLIDWSHPDYPSVYEGGRVPEEPGRANPFSSPAGGVQDE
EKWQRFLTFNNEQLRELLTNYGKVDLLWFDGDWERSAEQWNLPAFKNYLLSFNPEIMINSRLQGHGDYKTPEQGLPITRP
EGPWEFCTTINSSWGYVPDDNHYKSLHQIIRMFCDCISMGGNMLLDIGPKEDGTVDPRQEQILLGLGSWIRTHEEAVYGT
REGILTRYYLGGSTLSKDKKTLYLFVYDDPKESVCLKGLSSKIRTITVLHSGKELTYDIHGGVPWFQIPGTTWIHLTQED
THDYVTVLKIELEDELDMYGGSGAVVTHN
>lcl|NZ_CP026520.1_prot_WP_042229234.1_1454 [locus_tag=PC41400_RS07570] [protein=ABC transporter permease subunit] [protein_id=WP_042229234.1] [location=1673192..1674160] [gbkey=CDS]
MQAKASIRTTGTAGRPEAGTEKKPVWKRMVLNWELYVFIAPAFLYFLIFHYGPMYGILIAFKNFIPTKGIWGSPWVGFDH
FVHFFESYYFWDLLWNTLSISLYDLAVGFPIPILLALAFNEVRDGFFKRMVQTVTYAPHFISVVVMAGMIITFLSPSTGI
LVHLIEKLGFTAPQVLTDPNWFKTMYVFSGVWQSAGWGTIIYLAALSGVDPQLHEAAIIDGASRFQRIRHINIPTIVPTM
TILLILNMGSLFGVGFEKILLLQNPLNMEASDVISTFVYRSGLVNAQYSFSAAVGLFNSVVSAVLLVTVNQIVRRTSENS
LW
>lcl|NZ_CP026520.1_prot_WP_042229231.1_1455 [locus_tag=PC41400_RS07575] [protein=carbohydrate ABC transporter permease] [protein_id=WP_042229231.1] [location=1674179..1675087] [gbkey=CDS]
MSSTVKDTARDKLFLISTYVYVAIALILVAYPLIYILSASVSNPKYVGSGEMWLFPKGLTFEGYQIVFANTKIWMGYANT
ILYTVIGTAVNLLVTIPAAYALSRKDFVGRNFFMGMFMVTMFFGGGLVPTYLLVKGLGLMNSIWAIILPSAASIWNIIVS
RTFFQTSIPKELQEAAQIDGCTNMRMFLKIILPLSMPIVAVMALFYGVGNWNSYFSALIYLHEESKYPLQLILRQILVLQ
EMSAQTGGSLDASTASALNNKAEIAALVKYAVIIVSTLPIIAIYPFLQRYFVQGVMIGSVKG
>lcl|NZ_CP026520.1_prot_WP_042229228.1_1456 [locus_tag=PC41400_RS07580] [protein=extracellular solute-binding protein] [protein_id=WP_042229228.1] [location=1675124..1676734] [gbkey=CDS]
MKKVLKSGSLMLGLTLVIAGLSACGSADKAGEAGTGKNASEVKKDGFPIVDKPVTMTLMAPDVGVQDWNKMQVMQEMEKL
TGIKMQYQNAPIDSFESKKNLVFASDSLPDVFFGADIKPAEQVTYGSQGVLLPLENLIDNYAPNLKKILDEHPEIRKNMT
TPSGHIYALRTIDLSAVWYRSPMWYNGKFLKALGITKLPETTEELYAYLKRVQSEDPNGNGKKDEIPLTSVKLDDLRMFF
LGFWGIYDEVIYADKAGKVHYTPQEEGYKGYLTFLNRLWKDNLLDHETFSQTDEQKKAKGKNNRVALFNDYYPYFTMGTE
PDDSNPMMKPLKSEVPDSPVYGKHPGISSNGTFAITKSNPNPEAAMRWVDYMYGYEGGTLFAQGPEDVLWKYTDKEKHVK
EWKPVPGGGDREEYRGTLTPNYGILVPGAASSELSKGLKTDFDKWLDKENEEKLTPIAKIPFPSVFLTEEEQAEVTRLKS
DLDTYVKQMEAKFVTGQEPLANWDAYAGQLKKMGSDKIVGIYQAAYDRWNEGSGKK
>lcl|NZ_CP026520.1_prot_WP_156972815.1_1457 [locus_tag=PC41400_RS29680] [protein=hypothetical protein] [protein_id=WP_156972815.1] [location=complement(1676786..1676935)] [gbkey=CDS]
MRLLPAALPAGPLPDSSGAGLLIVARKGSRRVAAIVRLTKEEKPRRNAR
>lcl|NZ_CP026520.1_prot_WP_042229225.1_1458 [locus_tag=PC41400_RS07585] [protein=helix-turn-helix domain-containing protein] [protein_id=WP_042229225.1] [location=complement(1677178..1679430)] [gbkey=CDS]
MKSDYFKSKLFMRYIWSYLFILLIPLAFATIFIYRSAVSNLQSEIEQSHLNQLNQTKTIIDGRVKELIEIASRISYDERL
TLYKVHDPYYSREAISALDNYKATSSIIGDLFLYYHDDDKIYSSFGMSGLDVFSDKYSFHKWTQNGLRQDLNTLKFPKMR
PADLVSQPTSLQQSILAYLVPITPNSPAPHGTVMYFIPESELTGLIDSILGSYRGLTYIFDNNGQVLTNNGNEESLTAGD
LGALFKLPPGIHNQTLNGQSHSVVSVKSDKNGWTYVTLMPSAQFFSSVLHVRSFIILLVSIVVIVGAALALLLARMQYLP
ISNLVHFASSKSRSSRTSGNELERIRISLQEHSTRADLQEPYARNHALLMLLKYGQTKSLSADLLEAFDLKFDHDCHFAV
VIGFDKRSHLHDSKDDWQTMIQLFAEVDFPGLAARAYSVELPQPNQLALVVGFDPSGQTPFGQMQQIVEGIRTHLLELFD
VSPVIGVGTAYTSHAHLNQSYIEACSAFESNILTVHGSVTFFEKLSNAPERSTFWIPGNVLLKLSQSLKQGSYDVSEQTL
QEALKNLPAAGLSAPHLRLIHYDILTTLLKTASELGIHGLTQDVLSKLNFNSSEEFEAICLDLAGRICSQVERDRTKEEH
SLMDRIVGYIDNHFMEHTLSLEVVSAEYGISPSYLSRTFKEKTGINFIQYIWQKRMNEVMHQLKTTNDPLKEIIVRVGYL
DTPNFIRKFKKETGYTPGQYRTLNTSGEHA
>lcl|NZ_CP026520.1_prot_WP_042229222.1_1459 [locus_tag=PC41400_RS07590] [protein=alpha-L-fucosidase] [protein_id=WP_042229222.1] [location=1679589..1680986] [gbkey=CDS]
MSEMNAETQEGAAPAVVEGVHNYSREEDWAKPEDPALLERLEWFQDQKLGLMMHWGPYSQLGVVESWALSDADADWSRDG
IDWDVSAEDLKREYFGLNRTFNPLRFQPELWAGLAADSGFKYLNFTTKHHDGFCMWDTRTTDYRVTHPDCPFHAHRHADI
CGRLFDAFRAKGLAISAYFSKADWRSPYYWAPGMERGGRTSRGTSYDTREHPWLWEQFVKFTHEQIEELMTRYGRIDMLW
LDAGWVNAANNQDIRLGEAVEKARELQPWLLCADRTVGGPYENLITPEQTLPERPLLVPWESCITMGTSFSFRYEDTYKT
VRQLVHLLIEIVSKGGNLALNVGPQPDGRLPETAVARMKGLGAWLKVNGEAIYGTRICEPYSEGSWRFTRKGETVYAFCL
VEREQSPMEAEICIPLRDSFARVELLGMDMPLDYRLTEKGITVRLPESIREKEVPEAIVFRLSKK
>lcl|NZ_CP026520.1_prot_WP_042229220.1_1460 [locus_tag=PC41400_RS07595] [protein=carbohydrate-binding family 9-like protein] [protein_id=WP_042229220.1] [location=1681038..1682060] [gbkey=CDS]
MALSGVPEPRIVFAPKHYLCRRASGPLELDGRIDKPFWEAAEWTEDFTDIEGDAKPQPSKRTRAKMLWDDRYFYFAAELT
EDQIWATLTERDSVIFYDNDFEIFIDPDGDTHEYYEFEMNALNTVWDLLLVKPYRDGGPPVNGWDIGGLKTAVHIDGELN
NPDADNRKWTLEVAMPWSALRECAPENRPPVQGEFWRVNFSRVQWRTETVDGSYRKMIDPQSGNPYPEDNWVWSPMGIVN
MHYPELWGYVVFADGDGPESFALPDDERVKWELRKLYYRQRNHYERSGVFISDVGELLGCEPWTIVPQVETTARSFQICA
ASADGKAVYVIREDGKLWKE
>lcl|NZ_CP026520.1_prot_WP_174446800.1_1461 [locus_tag=PC41400_RS07600] [protein=transglutaminase domain-containing protein] [protein_id=WP_174446800.1] [location=1682068..1684902] [gbkey=CDS]
MNANTNDNPNRNNKPVKNRNEAENPSVNLLEDDFLKANLNADSEVERNAILRPDLSVFSMDAEALAEVQRKFDIKREFAR
NKEEELFGVFRQKLTEEETWALKFLFAYMPLNDLADYDGILFLNHVRTTLEIRAKVPWGERVPDHLFLHFVLPLRVNSEN
LEDCRGVLYEELAERTKHLPMEKAILETNYWCHEKATYVGSDLRTISPLTMLRSAQGRCGEESTLAVSALRSIGIPARQC
YTPRWAHCDDNHAWVEAWADGQWHYIGACEPEPVLDRGWFGAPARRAMLVNTRVHANYPGPEPVTLAEEWLTEINLLETY
APVRAVTVAVKDEEGRPVEGARVHFQLYNMAEMYPIAVMATDSEGLALFRTGWGDLLVRAVKDGLWAEEKLSAVGSSTLV
MTIRNAEEPPGTLDVDMVPPPEREGEALYAVTEEMLKLHQAREQEGAAVRKNFEDTFLGEQDAVKLAGELGLPPQRVWNV
LRKARGNSREIRDFLLERSPSNGEWPLRLLESMRDKDMLDTFRPTLDDHLLHALKWADAWEEDVFVPYILCPRVHFEMIG
PYRRLFEEAFTAEEAAAYRKDPSGLVRRLAAEYEVRDDLTNLPGRASAAGSFALKKGDRLSLDIMLVAVCRSLGIPARLH
PNNQKPQFRLDGEWNNAGWQEESKAQRGTLLLRPPGEHAHPGATEPRETGVHSDSAASEESAAAPAPEASYFQNLTVARL
EDGLYRTIVFPHGKSDLYAEPLELEAGSYRLTSGTRLKDGTVRVRFRHFTVLPGRQTELTPTFREAPSDVPVLGSVDGRI
VFGSLEENGETCVLEDLAGGRAVLAAWIEPEREPSKHLIRELTELSNLYAKAEAAVIMVVGGDRRTVSFDPARYGALPEG
THFVRDEDYGSLDEFNAKRPLPEAGFPHVFVVDRKGRIRYTQSGYKVGSAREALQIFENLCEND
>lcl|NZ_CP026520.1_prot_WP_042229219.1_1462 [locus_tag=PC41400_RS07605] [protein=glycoside hydrolase family 18 protein] [protein_id=WP_042229219.1] [location=1684918..1685979] [gbkey=CDS]
MSKPYVVAGYVADAGLPDVTGEDLGKLTHLNVAFGHVREDEIRIDHLACLDELTRFKTVNPDLVLLLSVGGWGSGGFSEA
ASTEEGRRRMAQSAVNILLAHPFDGIDLDWEYPCYSTAGIQSSPEDKRNFTLLLQTIREAIDEQGRKDGRRYLLTIAVGA
DQYYLDGTEMDQVHRYLDFLQVMTYDMRGGFQVLTGHHTNLYNSTGDLFRISTDTSINLYLSAGVPRDKIVVGAAFYSRQ
WEKVPDAANGLHQIAGGSGGYGPNYTTLANDYINKNGYVRYWDEEAKAPYLFNGSTFITYDDEESLQHKCRYVEEKNLAG
IMFWEYGCDKTRRLLDSLHRGLNKPIDSASKTF
>lcl|NZ_CP026520.1_prot_WP_042229217.1_1463 [locus_tag=PC41400_RS07610] [protein=alkaline phosphatase family protein] [protein_id=WP_042229217.1] [location=1686033..1686854] [gbkey=CDS]
MAKTIVLVLDGLRYDVMLSSMGYLNHLVEKSQAALYKVKSELPSLSRPLYEVLLTGTPSYINGITANHVVRLSRFPSVFH
LARQQGLTTAAAAYHWVSELYNSAPFDYVADRIQHDESKPIQHGMFYYEDSYPDSHLIGDAEHLRRTHNPDFLYVHPMNI
DDAGHKFTSDSARYRGQALHMDGLLAVLLPVWMEQGYQILITSDHGMNEDGHHGGTGDAERDVGLVAIGDRFTPGVYTDS
YLPQLAIAPLICSLLDIPVSAHMSDIPVAGFCK
>lcl|NZ_CP026520.1_prot_WP_042229216.1_1464 [locus_tag=PC41400_RS07615] [protein=SpoVR family protein] [protein_id=WP_042229216.1] [location=complement(1687171..1688601)] [gbkey=CDS]
MTPDEIRQLEHAIAEITEVAEGFGLDFYPMRYEICPADIIYTFGAYGMPTRFSHWSFGKTFHKMKTQYDFGLSKIYELVI
NSDPCYAFLLDGNSLIQNKLIVAHVLAHCDFFKNNARFSRTNRNMVESMSATAERVRQYEIEYGARRVEEFLDCVLAVQE
HVDPSLTTPGRMKPEEQDGKDNDKLGRNGRPKTGYEDLWDLERQDSEPEEEESRTVRFPPKPEKDILLFIEENAPHLEDW
QRDIMTMMRDEMLYFWPQMETKIMNEGWASYWHQRILRELDLTEEETIEYSKLNSGVVQPSRHSLNPYYLGLKIFEDIEK
RWDNPNEEEREKYGRRDGGGRNKLFEVREFDSDTSFIRNYMTKELVEDLDLYVFEKKGPEWKITDKTWKNIRDQLVYSRV
NGGFPYLEVYDGDYQRNGELYLRHSYEGVELDLKYVEKTLPYLYRLWGKPVYLQSVVEDKAVLFSYDGKKSHRKFM
>lcl|NZ_CP026520.1_prot_WP_042229214.1_1465 [gene=yhbH] [locus_tag=PC41400_RS07620] [protein=sporulation protein YhbH] [protein_id=WP_042229214.1] [location=complement(1688598..1689758)] [gbkey=CDS]
MTDSSFIVSREDWSLHRKGFQDQTRHQEKVRDAIKQNLPDIVSDESIILSNGKQVVKVPIRSLDEYRFRYNYQKSKHVGQ
GDGDSQVGDVLGVDPQPSSKGGKGEGAGDKAGEDYYEAEVSFEELQALLFEELELPFLQRKDKKNFSVKEVLFNDIRKKG
IMSNIDKKKTILENLRRNATQGDPGIHGISPDDLRFKTWNEVEKPHSNALIIAMMDTSGSMGSFEKFIARSFFFWMTRFL
RTKYEHVEIVFIAHHTEAREVTEEEFFTKGESGGTICSSAYQAALDIIDSRYPPEQYNIYPFHFSDGDNLTSDNERCVKL
IGQLMERSNMFGYGEVNQYNRSSTLMSAYRNIHHPKFMHYIIREKSEVHKALRSFFTPRTEGGELI
>lcl|NZ_CP026520.1_prot_WP_042229211.1_1466 [locus_tag=PC41400_RS07630] [protein=Dabb family protein] [protein_id=WP_042229211.1] [location=1689957..1690250] [gbkey=CDS]
MIDHIVLVKFGESTTEEQLQDVIARFKALKNHLSGVIDLQAGLNFSPNSQGYQLVLSVRFENQEALEAYGPHPEHQAVAA
YIREVGRVDSIVVDIRI
>lcl|NZ_CP026520.1_prot_WP_042229210.1_1467 [locus_tag=PC41400_RS07635] [protein=aldo/keto reductase] [protein_id=WP_042229210.1] [location=1690370..1691278] [gbkey=CDS]
MKKNRLGQSDLFISEVGLGCMSLGTEEKKAVELVCRALDLGVTFLDTADLYDEGRNEELVGKAIRGRRSEVVLATKVGNR
RVPGREGWTWDPSKAYIKSAVKDSLKRLQTDYIDVYQLHGGTLEDPIDETIEAFEELKEEGLIRHYGISSIRPNVIREYV
KRSGIVSVMSQYSVLDRRPEEEVLPLLEDKGISLIARGPVARGILTASGEEKAAQGYLDYSPEELPQVRGQLLEAAGPGR
DPGHMAIRYALSHPAVAAVIPGASSVEQLAHNAEAAAADPLSAEELEEIRRISKASRYAQHR
>lcl|NZ_CP026520.1_prot_PC41400_RS07640_1468 [locus_tag=PC41400_RS07640] [protein=cell wall hydrolase] [pseudo=true] [partial=5'] [location=<1691319..1691597] [gbkey=CDS]
LGINRAWANRSDFKGLRTIPQMIYQPHAFEAVQRLIPLPESPVADKPLACRPINGERSWPGDCPLTRYNQPLVGRYKLHC
FYEPTAKECENG
>lcl|NZ_CP026520.1_prot_WP_042229208.1_1469 [locus_tag=PC41400_RS07645] [protein=metallophosphoesterase] [protein_id=WP_042229208.1] [location=complement(1691815..1692852)] [gbkey=CDS]
MNRRQFLRSLAGAIAAASLGAAAFYTFFTRGASKELGSGLNAASPGPSGTPTPVPAGPGKLLASFWLLSDLHVTPYDSGT
SDKLKKALADLVKPEHPGDALILGGDLTEYGSDNDYKQLKNVLSGFKLPPLYANMGNHDYYDIWIDGKGSFATETQPNGK
TDAQSRQRFQAFMKTERPYGKFEAGRIPVFLLSQEAYVQEKPEVGEGAWYSDTQLDWLKAELAKLENHLPVLVMIHQPLP
QEGSDGGTHRVIRANELRAILAPYPNVFVFSGHTHRDFTANHYTRHATFHWFSNSTVGRSRSKVLSQGMYIQVYEHEVRV
RGREFSNASWIGAADWTVPLKKAAD
>lcl|NZ_CP026520.1_prot_WP_042229205.1_1470 [locus_tag=PC41400_RS07650] [protein=PLP-dependent aminotransferase family protein] [protein_id=WP_042229205.1] [location=complement(1692953..1694416)] [gbkey=CDS]
MNMRWKPRPYDHTPIFRQICTYFKEQILKGELSPGTRLPTERELALQLSVNRSTVSAAYEELRATGLVYSVQGSGTRVSE
SLWETTVQTAPNWDHYLGRRFFNPNAAITDKISEAASIPGIVNMIKGELSPDLMPLGLLGELAAQIDYTQAYSYFSDNRG
EPALRATLSDFMKTRSGISASPESILVTSGVKHSLQLISNTLLQPGDAVALEGPSYIYAMQVFSSSGIRMFPLPVDEHGL
IPDELPRLVHKHNIRMVFTNPTYQNPTGTTLSPARRAKLIDLCEQLRLPLVEDDPYGQLHLSGSPPVPPLKASPEGDKIV
IYLGTLSKISTPGMRTGWIVAPPPVIAKLAETKNRMGYSSSHSGERLAQLYMTSPSSVQSLISIREALTARRNAMLSALQ
SELYPYASVFGGNHNAPGGYYVWLKLNLPLEDKQLVDLAIKEGVMLWPGSIYGMKKGFFRLTYASIREHEIVEGVRRIRR
AFEKAVL
>lcl|NZ_CP026520.1_prot_WP_053228695.1_1471 [locus_tag=PC41400_RS07655] [protein=MFS transporter] [protein_id=WP_053228695.1] [location=1694559..1695884] [gbkey=CDS]
MRLLFKEKKYAWFLLAFLWVFGFVGSLGRFVMAYYQKDISETLGVGRSFLGATWSTAIFIGAISAPLGGWLIDKIGYKKV
MIYSAIANLLSLAIVLLFRNPIGYFLGFGLITGIAGIGASASYVLVANWFKHHRAKALMIIGSAGSLGLAVLTPVLVSNK
GWLNWTTVYTVLFIISLAFIPLILLIVRSNKEAPEEEAPAASVQDSSSPTNKEPGEEQPPAPAEPVFSLKKMSEGFKAFL
VYLTRPVMLVIMFALLTCGISMGTVEMNLMAIHQLAHVSDGMMSSSLSLLGLLELVGGVTFSFLLDYMPRVKALAYLYFF
RIAGFAVLFFHFDYSPILFSLIFGATYLGAVPGGILVAGEALNEGKKQIGLQTGILLLIHQLGGVVAAIAGGLNFDLAND
YQFLIAVNVVFSLIASAGYFVLYRKGIGTKKAPVLTGEVGA
>lcl|NZ_CP026520.1_prot_WP_042229202.1_1472 [locus_tag=PC41400_RS07660] [protein=hypothetical protein] [protein_id=WP_042229202.1] [location=1696144..1697103] [gbkey=CDS]
MKIIPRYDEDFFNCVESHYVSYWKAEGIPVEYMFYGALEDTSHVYEHFIAKRDNRWLFHDKKRVIDDIHHFGLTLRQERI
ETFQEAEPKIIEALDEGKIVFFFIDGYYFKHRTFTYLQKHEPHCMMISQLSHEGGKKWFIQDHSQPDFFDYYPHETVEKA
YDDNPTEQFVKEIMIFDIDRERAANPELDQIKAKFELWVRECRDDFKIYQQTIDFLEANSADISKVEEELEAYQNAFFFL
YSSRLLFSRFLDVMNIETSIRESLNESIQLTDRIRKLITNYKILGKISVPKITEACGKLYEIEPKMIQTLKEELLAQKI
>lcl|NZ_CP026520.1_prot_WP_009677159.1_1473 [locus_tag=PC41400_RS07665] [protein=phosphopantetheine-binding protein] [protein_id=WP_009677159.1] [location=1697119..1697370] [gbkey=CDS]
MNPIEHKIREILTQADIAVDQIGSNDDMTTMGLNSLTLIKSMLEIIKFYDLEIDEESVEFYNIKTIQNMVDFVSANVAEK
ANA
>lcl|NZ_CP026520.1_prot_WP_042229200.1_1474 [locus_tag=PC41400_RS07670] [protein=tryptophan 7-halogenase] [protein_id=WP_042229200.1] [location=1697416..1699110] [gbkey=CDS]
MEKTTGTAAHYDVIVIGAGIAGTILSAILARQQLSVLVIDAATHPRFVIGESMIPATSFMMRIMAERYDVPEIMHCSTLP
FVQNHITSNSGTKRNFSFFYHREGEAQNPKESTQLPVPNIPYGPEAHLFRQDTDAYMMAVAIKYGAKVKQKTMISDIDIH
EDGVNIKTTANEEFSAKYIVDASGFNSVLAKRLGLREEPSRMKTKSRSIFTHMVGVKPFDEITNKDEHKLPNRIHNGTLH
HIFDGGWLWVIPFDNNAKSTSILCSVGLQYDLNRNPDNVPTKKPEEEFADFLEAYPDVKKQFENAKPVREWISSGGRLQY
SASKVIGKRFCLLAHASAFIDPLFSRGLAITTETINSLAQRLIDAVREDDFNEERFEPVSRIIQQSYDTNDRLVSCSYTA
FRDFNLWNAWYRVWALGQAFTSLRLTQVLSKFYKERDMQILLDIEKAPYIGSLCVDFAEFQPLFEKIASTVEAVRDHGLS
TEEATKRIYGYYEEVQSFIPDQYEFLNPDRRFISKMDMLSLYEVVMWGAQQAPETIRKLYFDIDESFLKGTKEPASNEVP
VATV
>lcl|NZ_CP026520.1_prot_WP_042229199.1_1475 [locus_tag=PC41400_RS07675] [protein=cation:proton antiporter] [protein_id=WP_042229199.1] [location=1699132..1700406] [gbkey=CDS]
MHLLLNVMIALILILTSARIMGRVALLLRFPSVIGEMVAGILLGPTCFMYFFPEISKTVFNADVGNVIYILSNFGLCLYM
LLVGMEMKGIDRKSFRQAGMLASSGIIPPFLLGGGISLFLYASLAQQQITQLEFFLYMGVALSITSIPMLARILEEQGLL
QSRFASLTLLAGSIDDVVSWCILAVVIVMVQAKNMFSGMSTLLYTVLFVLIVLFVVKPLMERFGAKVQKAGFLSHGGLAL
VLLMTMGASFVTEYIGIYAVFGCFILGLAMPRSEVFLSEMNIKIRDITVVFFLPLYFAYSGMKTNLLDLFSADMLVPFLI
LLFCSIVGKYGGCTLYMRKIGFSWREASAVGGLMNARGLMELVVINIGMTYGIITPKLYAMLVLMALVTTALAMPIYSFS
MGTKPRSAAPVFPPKVQYRTEDGA
>lcl|NZ_CP026520.1_prot_WP_042229197.1_1476 [locus_tag=PC41400_RS07680] [protein=phosphopantetheine-binding protein] [protein_id=WP_042229197.1] [location=1700437..1700694] [gbkey=CDS]
MNIQDKIKQFIMEASDGNVTPEDLERVGDDIGRLGLDSLAKIKIVVLLEEAYDIETEVEEISPEVLNSISRLAEFIEEEV
QKVSV
>lcl|NZ_CP026520.1_prot_WP_042229195.1_1477 [locus_tag=PC41400_RS07685] [protein=class II aldolase/adducin family protein] [protein_id=WP_042229195.1] [location=1700736..1701455] [gbkey=CDS]
MSQFIEEMVYASQILYLEGHNDINNGQISYRDPESGNIWIRKGFLGWEETTPDDFILIDSEGNRLKGEGHIPSEWPLHTE
TYKARADVNCIIHSHPPYSMIFSATDLDLRPVTHDATPIMNTQRFTLTTNTVTNPNIAAQVAESLGKSNTLLLKNHGLVA
AGKTIPETVCWAAVLENACKLQLIAESTGVPYTWTEGHEIKMKSKIIFGATAIREYWAYWCRKADRHFNPDSKNHLVGV
>lcl|NZ_CP026520.1_prot_WP_042229194.1_1478 [locus_tag=PC41400_RS07690] [protein=AMP-binding protein] [protein_id=WP_042229194.1] [location=1701457..1704345] [gbkey=CDS]
MTNQLLTQEDVLSIFYNLKGREYVGSKFSKSPRPERHKNFEGPYTRLKTSVFNKTNEDMLPKTGLSILEQVLLSSFSVTS
VKFYGYQSEMLWSFPSAGGCYPVDVYVIVRSLDGVEPGIYYYSPLNTSLHKIACADSLFLLDEAILPQDQDADYYIMVSA
VPWRSCWKYSHRGYRFSFIDAGHVLANFQTIFHSLGVNFTTYTACKDQHIKTLLHLDSHEEPVGIIAVKQHETKPLAVQQ
AYRGNYQSRCITTFDSGLSEFDWEPIRLYQEKVRQSLQHQQPSEDWTVRHHIPAEWNDYNEMLRLIIERRSSSSYLPVEI
PRHDFIRLLEFVNSLGMPIRIFCVVHAVEGMIPGTYEFNGSELKMIKAGDFRARSTQLCFDQEFVHDSSVLFFFAIDKSQ
VNDKNFYTYQQRLVDMGILAQMIYLKSGELKLGYSAIGGYYDEEAKELFNLGENMDLLYSGTLGKDDVNSAFKVKMDRYY
LNKPKDSEEVKGDSPAQNGRIIDKLYEAVLKYGPKEAIRYADESYTYHQFWNQVEKTGAWIKQQIPKQYPVGISMKNCPN
YLFAYYGLLLEGYIPMLIDHTFTQDEVDNLCSYYNIGSLLTLDGEGRIQVQLRDENFDSFDEDEFANVATCRFSSGTTGR
PKCLMFTHDAVINAGVNWAKASGITSADNVQVTALFHNGLAFNTSLLAVFVSGACLFIAKQITPKSIWETVMKENISILV
AFPVVYDLLNNSKYCKPEHNLRLCLSSAAPLHYAVKQSFLDKTGMEICDYYGIVEAGPCTFNNGSKEASLGTPVPGVDIR
IVNEEGFAVADGESGIIQVRSSSMAKGYYKASFPFVISPDGYYQTTDRGFIESNHLHINGRSSDMINVAGKKMDPLEVEN
VLLKLSGVQDVAVVGIMNERKTTEYPVAFVVLDPGVTIDDLIAHCQQYLAPFKLPQKFVTVDSIPRSGVGKIKRKELADS
LS
>lcl|NZ_CP026520.1_prot_WP_042229191.1_1479 [locus_tag=PC41400_RS07695] [protein=LL-diaminopimelate aminotransferase] [protein_id=WP_042229191.1] [location=1704416..1705660] [gbkey=CDS]
MLVLQDYIQKSFADRIGGDKFGKGNEVYKFEKIKRAKAQALKEKPGIPIIDMGLGEPDWMADKEVVNVLSTEAAKRENRF
YSDNGSFEFKEAAAAYMERIYGVKGLDPTTEINHCIGSKPALAQIPAVFINPGDITLMTVPGYPVIGTHTQWLGGEVVKM
PLLEENGYLPDLNALSDEVKKRAKLLYLNYPNNPTGAVATRAFYEEVVRFAKENEIIVVSDEAYAALTFDGEPPLSFLSV
PGAKDVGVAIQSLSKAFNMTGWRLGFVAGNELVLKAFSYAKDNHDSGQFLAIQKAGVYCLNHPEITEATAAKYSRRHGLL
AEALGSVGFKVQKPKATFYLYAQAPKGIEDGPRFDSAEDFSQYLITEKLISTVPWDDAGAYVRFSVTFEADGEEEERALI
DELKSRLHGLRYIW
>lcl|NZ_CP026520.1_prot_WP_042229189.1_1480 [locus_tag=PC41400_RS07700] [protein=hypothetical protein] [protein_id=WP_042229189.1] [location=1705728..1706741] [gbkey=CDS]
MKTGTKVLPVEIPPIHGRMYYAFPLSILGPAGSYMPWMLNNFIQLQAYSLTRTREEQVAEIRFYNADEPGCFSEVLTIVP
EAAGKEPDIHRQAVEAIDRDQYVYAYVDKYYVADNPFHGKKHVIQEALIYGYDLEEQTFQILGFNKSRLFASSTVPFADI
AKGIVHSPVTDVFLLKPKEDFNPEFQLDKVIPLLTQYVDSADKHTGLALPERSQLVFGLDVYKSAVESLRCILDREGAVV
TNIAYLHILWEHKKVMGVRLQYMLEHGIAVHSGLDIVQEAYRQMEQELFNLRNKLLKYELTQSARLIEQIIQSLDAMALK
EKQVLELWLERLAMIPG
>lcl|NZ_CP026520.1_prot_WP_042229188.1_1481 [locus_tag=PC41400_RS07705] [protein=hypothetical protein] [protein_id=WP_042229188.1] [location=1706775..1707776] [gbkey=CDS]
MTYNQSLRPFNDFWMNCVSNTVYSIAVSNDSSYRTAAYLNHYTYQVAVDPSFTYLTMDHPLKYQNEVFNDFIREWEPYYF
RNPQAIGEEIGELLDSGQLPVVHVDLFHWIPNSLAWKQYHWYHYSLITGYDAERSVYRVLDDDINGFRDFEVPPERLQEA
FLSSQTENNPAYQEPPCSLIRLKDQIPPFEFKLSDLSRHAERLIGELDNLQTEHLWEGTPEPDKFEESMTNGLVGVNIIE
SRQTGNRLMLRYLLDAGFIREADFQTLDKELEDLALGWFKAKNVFIKAKFSLKRRFNRDQLLTLAVPLLSRERSFWAALL
ETTGAAAALSKES
>lcl|NZ_CP026520.1_prot_WP_042229187.1_1482 [locus_tag=PC41400_RS07710] [protein=hypothetical protein] [protein_id=WP_042229187.1] [location=1707804..1709783] [gbkey=CDS]
MKYIGTEQTEASRHHGRLQPVPGVRHYQAVRACRLKPDLEGGPGWTFHNAPMLAFWRQRFYLQYITCPVEENHPPARTMI
TSSADGKEWTEPVIAFPSYEIPEGVYSGPDGADLPKGSLAVMHQRMGFYTAADGRLLTLGFYGICPVLSVMPNDGRGIGR
VVREIYPDGSMGPIYFIRLNVHAGWNTGNTSYPLYTESPDEGFVEACRLLLQDRLATIQWWEEDQSEDGFYPLAGYKALS
CYPLPGNKVAGIGKNGKYALSEDNGESWTETGEIDGLVTAGSKLWGQQTPDGKYALLYNPSPNNAYRWPLAAAVSEDGLH
FDKLRLIGGEVPPRRYTGFYKFYGLNYVRGISSGNGTPEEDAIWVTYSMNKEDIWVARIPVPLVPDFHDDGDAAAPTADT
DTDTDTDTDASTLGGRVYGLSGSPIETAKAAAPAPDTDGSGPVSAAGQLCEMPALSSSGLEDFDGWSIYRPKWSEIRLRG
DSGLILSSRDPYDYVKVEKAFAFRAGHQTDVTIGIRPLAIGTVPLTIELTDGRGETAVWLHWSTDGRISVLQEESRVFLQ
TYEEGAEYECVLSIDPENGSLSLSIDGTPAGKLFEYHAAKAGTVAFKVGERPFRVSLDTPLDGGDLGIPDEPLLPSAYCI
RYVKLQSSRYSNRVQAERR
>lcl|NZ_CP026520.1_prot_WP_042229185.1_1483 [locus_tag=PC41400_RS07715] [protein=CoA-acylating methylmalonate-semialdehyde dehydrogenase] [protein_id=WP_042229185.1] [location=1709793..1711238] [gbkey=CDS]
MSILLKNYISGSWQESSSSVTETVYNPANDGVLAQVPLSTRKDLDEAVQAAKAAFLPWKNTPVQKRARILFKYQQLLVEA
WDELAALITRENGKSQQDAYMEVQRGVECVEFAAGIPSLMMGQITPDVGTGVETTMLRYPIGVVGGITPFNFPMMVPCWM
FPMAIACGNTFILKPSERTPLLANRLAELFTEAGLPDGVFNIVHGAREVVNGILEHPEIKAVSFVGSQPVAEYVYKTAAQ
HGKRVQALAGAKNHSIVMPDADEELTVKSIIESAFGSAGERCMACSVVVLVGGDRDDLKQRLIDAADRLQIGNGEDKETE
LGPLIRQSHKDRTLDYLSSGEQEGAVLLRDGRKDGAASGAGYFLGPTVFDEVTPAMKIWQDEIFAPVLSIMQADSLEEAI
AIANRSSFANGACIYTGQLAAVRQFREEIDAGMLGVNIGVPAPTALFPFSGYKNSFYGDLHVNGKDGLEFYTRKKVMTVK
A
>lcl|NZ_CP026520.1_prot_WP_042229183.1_1484 [locus_tag=PC41400_RS07720] [protein=MBL fold metallo-hydrolase] [protein_id=WP_042229183.1] [location=1711279..1712280] [gbkey=CDS]
MELLKQETEEIVSVKLPVGSVLQWLNAYLIRGEDGYTLLDAGMPDEETFGLWQQAWRHLGISPRDITSIIITHHHVDHYG
LAARLQEVTGASVRMTESCFLQASRIWSADSVYDEEVLELFARHGAPRHLLEVTAPELKHYVKSVNVHPDAEFIRAGDKL
ALGKSIFEAELVSGHAEGHLCLYDKQEQLIFCGDHVIPKMLPNLNFTLGFDENPLGSYFSSLKEHAELPVRKAYPGHMIP
FRKYNARAADIAAQYRQRIDDILARMDKPMTAFELCCLVFFVPVNAEQLRFKLFEMLACLNYLQSLNLIKPAEAGADGRL
YYRRAPAHANRNP
>lcl|NZ_CP026520.1_prot_WP_042229182.1_1485 [locus_tag=PC41400_RS07725] [protein=NUDIX hydrolase N-terminal domain-containing protein] [protein_id=WP_042229182.1] [location=1712295..1713092] [gbkey=CDS]
MEANKIVDWAHYIFVSAKAGLTYASDSCDLKRYEVIKGVAQEMLSLYTEPHEDLPRRVPNGENKIYLTFDDGPDPVYTPQ
ILDLLKKYGITSTFFLMGERVRKHPHIVKRIVDEGHAIGNHTYTHPFMILLNEKELNEELSLTDEAIRDAVGFAPVLFRP
PYGLLNQGTIDLLRARGYEIMMWSEELEMYDWSFPGVDNMVRLIEEKTESGSMILLHDGGGTREETVRTLDKVIPLLLER
GFHFAADYLEQSGNEHGRTAAETVR
>lcl|NZ_CP026520.1_prot_WP_042229179.1_1486 [gene=bluB] [locus_tag=PC41400_RS07730] [protein=5,6-dimethylbenzimidazole synthase] [protein_id=WP_042229179.1] [location=complement(1713196..1713843)] [gbkey=CDS]
MFSSEEREALYKIVYTRRDVRTFRSDPIPDETVIRLLEAAHHAPSVGFMQPWNFILVESDEIKERLAWAADKERRALAIH
YEGERQDQFLNLKIEGLKQAPLTVCITCDPTRGGSHVLGRNSIPETDMMSVACAIQNLWLAACAEGLAAGWVSFYKKNDI
RDILAIPPHIDPLALISIGYTDHYPAQPILETANWEKRRELDGLIYRETWGQRGN
>lcl|NZ_CP026520.1_prot_WP_042229177.1_1487 [locus_tag=PC41400_RS07735] [protein=ABC transporter substrate-binding protein] [protein_id=WP_042229177.1] [location=1714222..1715538] [gbkey=CDS]
MKKLMVLMIALAMLLSACGGGGGNTASTDNKGSGGDNGKPVELQFYFPVAVGGPITKIIDGLAQDFEKENPNIKIKPVYG
GSYQDTMTKVATAVQGNNSPDMAVLLSTDLYTLLSMKAIEDLTPRFSKDYFGNFYEAFLGNTKSGDTVWSVPFQRSTIVL
YYNKDMFKEAGLDPEKAPKNWNELREAAAKLTKTDASGKVSQWGVEIPSTGYQYWMLQALSLQSGDNIMSSDGKQVFFNK
PHVEEALQFYMDLGQKDKVMPNGAIEWATVPSDFISGKTAMMFHTTGNLTKVKTDAKFNFGVAFLPANKQFGSPTGGGNL
YVFKNIPEERKKAAVKFIEFLTKPERVAQWSIDTGYVGTTKAAYETDQLKKYIESFPQAKVARDQLEYADSELSTYQNGQ
ITKLFNDNIQAALLGKMTAKEALQKSQEAADGILKNFK
>lcl|NZ_CP026520.1_prot_WP_042229175.1_1488 [locus_tag=PC41400_RS07740] [protein=sugar ABC transporter permease] [protein_id=WP_042229175.1] [location=1715701..1716597] [gbkey=CDS]
MAQRTATIKNKFNWKENAFAYALLLPSLIFLALFTFYPTLKSIYLSFYSSNFESSTFVGLDQYKSVLQDDVFRKVIKNNL
LLAVGTVPTSIFLAIYMAIWVNNKLKGNGLLRAAFFYPTIIPMIAIANVWLFIYTPDYGLLDKFLGVFGIDSPNWLGSPE
WVMYAMMLMVLWKEAGYFMIFYLAGLQSLPKEVYEAAKLEGASPFQVFRKITFPLLMPTTTFVFIIAVTNSFKLVDHLYI
MTKGGPDNGSNLLLYHIYETAFSFWDMGKASVLTVILLVILLAVSIFNYAYLDKKIHY
>lcl|NZ_CP026520.1_prot_WP_042229173.1_1489 [locus_tag=PC41400_RS07745] [protein=carbohydrate ABC transporter permease] [protein_id=WP_042229173.1] [location=1716618..1717430] [gbkey=CDS]
MHQVINRTVIILLAIAFLIPLVWTVFTSFTPSEEIITRANPFAIENPTFVNYINAWKTIPFLQYYWNTIVIVVGILAVQL
VTITLAAYAFARLNFVGSGILFIIFLTQIMIPPDILIYPNYTIMKELNLVDTKLAIMLPYWASSFGVFQLRQTFKQIPLD
LDEAARVDGCKWWQTLWYVYLPAAKPTYIAFGLISVSTHWSNFMWPLVVTNSVESRPLTVGIAIFAQSFETGAQWGTVTA
ATLMVILPLLIAFFVFQRQFVDSFMHSGIK
>lcl|NZ_CP026520.1_prot_WP_042229171.1_1490 [locus_tag=PC41400_RS07750] [protein=MBL fold metallo-hydrolase] [protein_id=WP_042229171.1] [location=1717445..1718215] [gbkey=CDS]
MELILMGTACASSGRDRDNTYLLVRDTSGSTLIDIGGNPLGKLKTLGLTTHEVKRVVFTHVHIDHIYGLPSLLWGMWIDG
REDPLEIYCDESEKEWLQNWLDLMKLTKWPIAFEISVRAFQWEKPALLWKEGETALSVFPSRHGVAAVGIRVECGGTVFV
YSTDTMPNPLIRDMDRIDLLIHESTTAERSLAMHSTLRDIAGYYDWSKIGRGILVHLTDDEPYEDLLKEIPEPVSSKIFL
GRELERIFLSGENSID
>lcl|NZ_CP026520.1_prot_WP_042229170.1_1491 [locus_tag=PC41400_RS07755] [protein=HAD-IIA family hydrolase] [protein_id=WP_042229170.1] [location=1718288..1719079] [gbkey=CDS]
MDHYAGYIFDLDGTIYLGDHAIEGAVETIHHLQSKGKKLLFLTNKTIESRENYVKKLHEFGVKIGMEQMLNPALVTIRYL
QNHYPGQKVYVIGEPILKEEFLENGISFADTPEETGVVVISWDRDFHYNHLDFAYQAIKHGAEAIATHPDRTCPMPGGDV
PDCGGMIGAIEGTTGKKVTMIMGKPSVMTALTALEILQVEAKDCLMTGDRLETDIRMGAEAGMSTALVLTGITSEADVAS
AEFKPTYVLPSVHAIVKQGCSSQ
>lcl|NZ_CP026520.1_prot_WP_016423882.1_1492 [locus_tag=PC41400_RS07760] [protein=glycerol-3-phosphate responsive antiterminator] [protein_id=WP_016423882.1] [location=1719274..1719876] [gbkey=CDS]
MKETTERAQRVAYCLKKKPVITSLMNAEELPGILQTSSNIVFILKSDIFMIEGIVEKIREAGKLSFVHFDLIEGIGKDKA
GVAYMAEKIGIDGIVTTKNNIIAEGKKYGLITVQRLFVFDSVSLDNGIKMTKSSQPDAIEVLPGMVCSRIMKRIRAEVDV
PVIAGGLMTDLEDFNAALGSGVIGISTSSKELWRWQDENR
>lcl|NZ_CP026520.1_prot_WP_042229169.1_1493 [locus_tag=PC41400_RS07765] [protein=lamin tail domain-containing protein] [protein_id=WP_042229169.1] [location=1720093..1721994] [gbkey=CDS]
MRKSLRKKMAGVLTLALAAAPLLPVLPTQSVQAAAMPKLLITELVPDTTNFASYDAFEYIEVYNNSAVQVDLQGYRFKAG
SWNAQIAQSYKLGPWETGVVWTRRAEIAPLGKEAFNSYYSLSYASKYVPDSKLHIIENVGGLTNSGTQTVTILDPAGAEA
VKASYTADDVAEGKTITYRYPAAGGTAMQKIAGLQAPTPGRLLAGQAPARPKQDNQAPQAPAGVTAVASGGSAKLAWSAN
PEADVFQYNVYQNGVLLYTVPASQREFTAYSLIGNKPYTFQISAVDLSENESAKTSVTVTPSHQLITQEERAVNPKDSKY
QSLWNISSDGPVVPGLKQDLVPQGMAYYGANNWLLTVAYLEDGRPATLTVTDASTNQYVKSVVLYNSDGTPYTGHAGGVA
VSRDHVWIASEGALHQLRLSDVTGAQNNGEVSFIGSVPVPVDAAFNTFADGVLWVGEFYEAKSYPTDPSHKLVGRDGVQH
YAWTAGYRLDPVTDTIRSDKWNGSAGTAAVPDYLLSITEKIQGIAFMQNSVVLSQSYGRGNDSTLYRYNNPLQEPAHATG
TVGGTSVPVWFLDGQSAKATNSKLTAVPMTEGIVPVGDDLFVLFESGANKYRYTTTYIMDRILKINWNQWDQM
>lcl|NZ_CP026520.1_prot_WP_042229168.1_1494 [locus_tag=PC41400_RS07770] [protein=hypothetical protein] [protein_id=WP_042229168.1] [location=1722113..1722292] [gbkey=CDS]
MLIAVGLVMLLAPGAVWTVKESWKSNDGTEPSDLYLLSTRLGGGIFLLVGIACIVVSRL
>lcl|NZ_CP026520.1_prot_WP_042229166.1_1495 [locus_tag=PC41400_RS07775] [protein=hypothetical protein] [protein_id=WP_042229166.1] [location=1722329..1722748] [gbkey=CDS]
MIKKWNGTLFGTLGLTVLLSFLHGAGGELLFLSAYKYSAIVESLGLAVAVLTIFYALPVFACFRTKYWAVLAFLFVLSPL
GSLLFFFIGGLFFSVAEDDLGAGILWFITTGINLVSVVLGTLLGGLTNLMLHSRRMLNS
>lcl|NZ_CP026520.1_prot_WP_042229164.1_1496 [locus_tag=PC41400_RS07780] [protein=HAD family hydrolase] [protein_id=WP_042229164.1] [location=1722777..1723442] [gbkey=CDS]
MPIQAILFDFDGTLADTLPLSFDAFRCVFETYENETLTNEQITAIFGPTEDDIIRRNVSDSSKAEAAIADYYAFYENHFH
ERVSMPRDIVHMLEDLNGQGVRMVIITGKSRRALDVSLNKLGIEKYFCFSVSGDEVERPKPDPEGIFAVLKLLGLSKEEA
IFVGDSSADITAGKSAGLMTIGAHWFETVQTARFDPEPDRVLKSARDLTELVRESGAALER
>lcl|NZ_CP026520.1_prot_WP_042229161.1_1497 [locus_tag=PC41400_RS07785] [protein=nuclear transport factor 2 family protein] [protein_id=WP_042229161.1] [location=1723469..1723840] [gbkey=CDS]
MTYQNALEAYITATNTHDFDEVRKILHPQAVYWFTDKSCTTPEQIRAYFENTWNLIREEVYSVRDVQWIAEGPQSAACIY
TYCWEGYHEGKFVSGSGRGTNVFVKDAAGEWKLVHEHLSGAVQ
>lcl|NZ_CP026520.1_prot_WP_042229159.1_1498 [locus_tag=PC41400_RS07790] [protein=hypothetical protein] [protein_id=WP_042229159.1] [location=1723923..1724549] [gbkey=CDS]
MGRNFANLHIKSNNLEKALKALRELSEGSPELLGKKSSVQNRAVSDADKTGNEAQDNTFVFYVSKSNENWISVLHDYFVW
GTVKKIGKALSRLIEEPVVTIGFINDEIFELSIFENGDIGAERIFCGPWTREDYELQEERLHDDRLREVLDIRKEDFDEF
IKITSPEQAVDQLSEFLSMSLWSDSEWLPHDENLRNRFEKYEFHIENT
>lcl|NZ_CP026520.1_prot_WP_042229157.1_1499 [locus_tag=PC41400_RS07795] [protein=DoxX family protein] [protein_id=WP_042229157.1] [location=1724967..1725389] [gbkey=CDS]
MSTIISAETTVKEETRHRGKTIAYWTFTLLLVISITLSGIGQLMRYGGNVDLVKNIGFPLYVTSILGTWKLLGVIAIIVP
GFPRLKEWAYAGIFFLMTGAALSHAFADDYGDYGFNLILPLFYAALNIASWALRPGSRKM
>lcl|NZ_CP026520.1_prot_WP_042229155.1_1500 [locus_tag=PC41400_RS07800] [protein=hypothetical protein] [protein_id=WP_042229155.1] [location=1725519..1725983] [gbkey=CDS]
MSYIVDFKNVSTVGLESSPVAEALAGLRANEARYFMNKYKHEFTVVPASESQENLDYVNRILKEERGLAFAAKPLETSQF
QVENIQFTYVFYEDGLALNVMYTVDDPKKRAVGFKLSEGMEVPQELEGKFKFARQKSKLAGTIRGSFFVIKGEY
>lcl|NZ_CP026520.1_prot_WP_042229152.1_1501 [gene=kduI] [locus_tag=PC41400_RS07805] [protein=5-dehydro-4-deoxy-D-glucuronate isomerase] [protein_id=WP_042229152.1] [location=1726392..1727222] [gbkey=CDS]
MEIRHTTNPADFKTFTTERMRNDFLIENLFQEERISMVYTHYDRMVIGGAVPVKETLELEAAETLKTEYFLERREVGFLN
IGGDAVITADGESFELGRLDALYVGKGNKKVTIASKDSSNPARVYFCSALAHAEIPSRKISIKEANPNHLGSLETSNERI
LNQYIHADGIQSCQLMMGVTQLKTGSVWNTMPAHIHDRRMEAYLYFDLKEDARVFHMMGEPSETRHLVVSNEQGVISPSW
SIHSGAGTSNYSFIWAMAGENYTFKDMDMVPMNELK
>lcl|NZ_CP026520.1_prot_WP_042229150.1_1502 [locus_tag=PC41400_RS07810] [protein=DeoR/GlpR family DNA-binding transcription regulator] [protein_id=WP_042229150.1] [location=1727261..1728010] [gbkey=CDS]
MLAAERHRTILEELEGRKSVKVSELSDKFQVTEKTIREDLEKLEEKGLLKRTHGGAVPLPDEESLLPQIPNIKHVREKSA
IAEYALTFIEPHDIIALDAGSTTLEIARKVKNMPLTVLTNDLLIIRELTAKDQIRLVVPGGYRHHNLLVGTDPDWVKKLN
VHKLFLSATGIHPDYGLTIFTEELTRLKRVYMECAKTIYCVADHKKFGKGALITFAGLDEIDYLITDSGIGSETLSKYGE
AVNILQAPL
>lcl|NZ_CP026520.1_prot_WP_042229148.1_1503 [gene=kduD] [locus_tag=PC41400_RS07815] [protein=2-dehydro-3-deoxy-D-gluconate 5-dehydrogenase KduD] [protein_id=WP_042229148.1] [location=1728040..1728795] [gbkey=CDS]
MNLFDLSGKKAVVTGAGRGLGAAIAIGLAEAGADVALVTNRTPADDVKARIESLGRRAVTIQADVSDRSLLSGVIGQALE
GLGGLDILVNNAGIIRRTPAAEHDYQDWQDVLDVNLNSVFTLCQLAGRHMIAQGSGKIINVASMLSFQGGINVPGYTSSK
HAVAGLTKALANEWASKGIQVNAIAPGYMSTDNTEALRDDPVRSKQILERIPAARWGTSQDLVGPAVLLASRASDYMNGH
ILCVDGGWMVR
>lcl|NZ_CP026520.1_prot_WP_042229146.1_1504 [locus_tag=PC41400_RS07820] [protein=MarR family transcriptional regulator] [protein_id=WP_042229146.1] [location=1729078..1729548] [gbkey=CDS]
MEEQDRIHRIFQSYREVNQAFFQVLMKAAQLYGLTPVQLLTLKNLKEHPLIGLGELSEYILLGTSTTSGIVDRLVKAGLV
SRERSETDRRSVTLKLTEQGLEQWNLVERDHMDRLYILASVSEEDQRQLERINTQILEILQKVRDEETHEYRSTPI
>lcl|NZ_CP026520.1_prot_WP_042229145.1_1505 [locus_tag=PC41400_RS07825] [protein=DHA2 family efflux MFS transporter permease subunit] [protein_id=WP_042229145.1] [location=1729523..1731034] [gbkey=CDS]
MSTEAPQFKRGPIVASLLIGAFVALLSQTFLNVALPGMMKDLKVSENTIQWLSNGYMLVSGVLVPISAFLIERFTTRKLF
IAAVGLFTLGTIISGFAPGFELLLTGRLVQAAGAGVLMPLMSIVFLTIFPVEERGKAMGMMGVAMIFAPAIGPTLSGWIM
EHYSWRALFYIIIPLSAFALIFGAVSLKNVTRTSKPRLDILGVILSTLGFGGLLYGFSDAGKDGWDSTIVVTCLAVGAVS
LILFIWRELVTEKPILEFRVFKFNMYSLTTVINVIVTMAMFSGMILLPLYLQNIRGFTPFESGLLLLPGAILMGIMSPIT
GIIFDKVGARWLSVTGLLITAVTTYEFSKLTVDTTYTTLILIYTIRSFGMSMLMMPIQTAGMNQLPQRLNAHGSAMSQTL
RNVAGAIGTALLVSLMTNAAKSHGAELVAAGGINPADKVKMAEIAQQSAIYGINHSFVVATWMTVAALVMAFFIRKVKPH
VEIKHESVVIENKDAEAVTVSSK
>lcl|NZ_CP026520.1_prot_WP_042229144.1_1506 [locus_tag=PC41400_RS07830] [protein=hypothetical protein] [protein_id=WP_042229144.1] [location=1731185..1731490] [gbkey=CDS]
MKCSICQWVKIVDMNNEAMTEQLFVHGEIEGAALTVGASVVTHSLGLKKFEVVYDKREGIETARFKVVDIEVDMLQHPFT
TRAYLEPQVLIIGQHDVGETE
>lcl|NZ_CP026520.1_prot_WP_042229143.1_1507 [locus_tag=PC41400_RS07835] [protein=DUF423 domain-containing protein] [protein_id=WP_042229143.1] [location=1731610..1731981] [gbkey=CDS]
MMRLFMMLGSVNMFLAVALGAFGAHMLKTKISADMLAVYQTGVQYHMIHAVGLIAIGLAADRLTQNQSLVNASGWAIVIG
IILFSGSLYVLSLSGIKVLGAITPFGGVAFLAGWVMLAIAAMK
>lcl|NZ_CP026520.1_prot_WP_042229142.1_1508 [locus_tag=PC41400_RS07840] [protein=PAS domain S-box protein] [protein_id=WP_042229142.1] [location=1732197..1733687] [gbkey=CDS]
MYKTFSPDEDSESGFIAPAMIPVFAYAFKEALIGMAVVNQQGDILEVNEVLCGMLGYTRRELLGMNKDDIVCAAVGTGDW
MSRWKELAATKKLQMEKRFRHKDGHWVWAMLNVSLLPGTDGSFVYYAQLQDLTSRKAMEEKLERSRQELGPLFEYNPDIV
CIVDMKGHIVSINSTMRQITGYETGELAGHHCKRFIFPGDLVAAAELFEAAKSGRTGTGEFNVSRKDGSWITVEVNTLPI
QVGSHVEGIYAIDKDVTRRRETEERLRTSEKLSAVGQLAAGVAHEIRNPLTALKGFTQFLQSGAEGKQEYYEIMMSELNR
IDLILSELLMLAKPKASEFAWLNPELVIRHVLALLEAQAILKNVQFLTNVDGSLSGMMGAENQLKQLFVNLLKNAIEAMP
DGGLITIRLQEQADKRICFAIADQGSGIPQDLIDKIGVPFYTTKDQGSGLGMMVSRKIVEEHKGTLQIESVIGEGTIVYV
RLPINRESGEMEEKRK
>lcl|NZ_CP026520.1_prot_WP_042229140.1_1509 [locus_tag=PC41400_RS07845] [protein=EAL domain-containing protein] [protein_id=WP_042229140.1] [location=1733684..1735579] [gbkey=CDS]
MIPVLLGGYAAVIVIGSVLLRAYAVWICLGASAVLAGALYTLNKRGNLAASGQEGLPALDNKPADPPIADETSRLLDTLF
DYSPNAISVTDEHGKLTRVSRTMERLLGATNEQLIGNSFGAYVSDEHKDKLKQRYGDLRTGTPQLFDIPVLHRSGYRIDL
SATGVPLETEGANRRFLFMFQDITERRRADEQFRHMAYYDDMTGLPNRRLFKEQLTETLHLAESHHQRVAVMYVDIDGFK
LFNDCFGYDYGDMLLLQVAERFTRCVSDNDFIARTEGDEFAFFYTGIQGREQALQLASRLLSVLEEPFMLEQYELHVTAS
IGISLLSREHEDADTLMKHANMAVTRAKENGKSDIQIFNSEMRSVSLQKLQLENDLRRAITNNEFVLHYQPQVEIESGQI
VGMEALIRWNHPQKGFIPPGDFIPFAEQSGLIVPISEWVLHEACQQNQAWQDAGLPKVPVSVNLSSRQFLQHNLKAKVDQ
VLQHTGLAPQFLELEITESMTMDVEHAIASLLELKQLGVQVSIDDFGTGYSSLSYLKRFPVDRLKIDRSFVRDIMEDPSD
AAIVATIISMTRHLNLKVIAEGVETEEQLSFLHRNQCHEVQGYWFSPPLTADKMGSMLRRITCSAGRLKEA
>lcl|NZ_CP026520.1_prot_WP_241688815.1_1510 [locus_tag=PC41400_RS07850] [protein=hypothetical protein] [protein_id=WP_241688815.1] [location=complement(1735806..1736345)] [gbkey=CDS]
MTTNKNQTSKQGHLFTVKILMEEETNGLALQKLLQVLNNANFTDFTIEQGIELGKLIALNTPSSGTSSTGPSSQAAGSDG
TPASSGNQAASASKTGSSAPSTASASNQTRPQVEEQLRKMIKDNTLIRLSVVKGAGIRLNLPCRVLNFDEESGNVTVYHV
DEKKVYLFKLNEIDNMSLH
>lcl|NZ_CP026520.1_prot_WP_009677149.1_1511 [locus_tag=PC41400_RS29685] [protein=hypothetical protein] [protein_id=WP_009677149.1] [location=1736483..1736629] [gbkey=CDS]
MMKVEPIELDGNTAIAIEVKLPKTTLLVVTTDKGYIMCGAQFIPYFTG
>lcl|NZ_CP026520.1_prot_WP_042229139.1_1512 [locus_tag=PC41400_RS07855] [protein=hypothetical protein] [protein_id=WP_042229139.1] [location=complement(1736861..1737655)] [gbkey=CDS]
MTEEINKGKHEKSSNGLIGEALIVILITAFGYLAGYLYEVGYKDYYELPDMFVEVGLPTIIKSTWSLFRSIMITLLFINI
LSYLETRRVEISLSYISLMGYIGAFSILYNLGSIGTLAYGLFFAFTIIAYVIIPLVKYREIKSWIDRIFYTEKTGRNNIF
KNISQKYGTKVTVIFLFFAIISCNMFLVQEKGKNDAREETEYLVTNTTEPMIVIDSYKDYFILAPVDIKNKTFTPQYKLL
KQSKDTPDLPTFEKKKIGPLKPKE
>lcl|NZ_CP026520.1_prot_WP_042229138.1_1513 [locus_tag=PC41400_RS07860] [protein=hypothetical protein] [protein_id=WP_042229138.1] [location=complement(1737942..1738370)] [gbkey=CDS]
MNLPFAIAIFINLKGKTELYYGETEEIENRLSNIHADFNYFQNEEQLIAKLKIISETKEYKKIKFTPITSKCYTISLHWI
FQMYVDDQTQIYYGEKESARQAVTNYCKGKHVYGKGYPSKFQMLWQLEYYKTAKSNTVYIPV
>lcl|NZ_CP026520.1_prot_WP_053228694.1_1514 [locus_tag=PC41400_RS07865] [protein=helix-turn-helix transcriptional regulator] [protein_id=WP_053228694.1] [location=1738510..1738956] [gbkey=CDS]
MDFGKYLKQLRGNMTLREAADRSGLSHTYISFLEKGERPGTKKPIQASPEVLKKLAMAYNHPYEDLLQRAGYLDEQLMMD
YEGKIYSEPVMASFVREPVSKYSYSNVIELSTLLFSKNNIAYEGRILSDNEKRKILNIVEVLLDDEEP
>lcl|NZ_CP026520.1_prot_WP_042229136.1_1515 [locus_tag=PC41400_RS07870] [protein=hypothetical protein] [protein_id=WP_042229136.1] [location=complement(1739251..1739493)] [gbkey=CDS]
MKKVYSVAIVQKVVRNNLVCEMHYGESTAVQVALVNVKMPFFKFFDSAQEMYTELNKSKLNMVFDRVTTHHVNTIYAAAI
>lcl|NZ_CP026520.1_prot_WP_042229134.1_1516 [locus_tag=PC41400_RS07875] [protein=helix-turn-helix transcriptional regulator] [protein_id=WP_042229134.1] [location=1739640..1739906] [gbkey=CDS]
MVIVKPCLNKLLAERNLKQIDLATMTGISAPVLSRFDKNTQHVDNHLFTIAKALKVKIDDLFEIVDNDDLFETVENNEGM
LKVDINEN
>lcl|NZ_CP026520.1_prot_WP_241688816.1_1517 [locus_tag=PC41400_RS07880] [protein=hypothetical protein] [protein_id=WP_241688816.1] [location=1739992..1740588] [gbkey=CDS]
MYNKIIITNQLRGYFMTPFGIFFVVSLSVFIISVAIINIKKNIKAAIPTSSFDGKHVLGIPGIGTGISAKVVTNKDNITI
NNTQIIPMDRFQFARGVSAQEIVLDDQQKSVILRGIAGGLLLGPIGAVVGGMSGLGTKQQKKEVTQHYIQLTYTNKDNEQ
IDALFLVDRDKKYCDGVADYINKETGKTAPVQKLQYEI
>lcl|NZ_CP026520.1_prot_WP_272509685.1_1518 [locus_tag=PC41400_RS07885] [protein=type II toxin-antitoxin system PemK/MazF family toxin] [protein_id=WP_272509685.1] [location=complement(1740625..1741008)] [gbkey=CDS]
MSKLLLDATCDDGKMVNRGEVWMVKFDGIGSEQRGLRPALVISNNRGNKYGSTVIVCAITAQVHKAKLPTHVVLKASPNG
TIKDSVVMLEQVRTVDKSRLLHKLARVGPITMGKVTEAHAVSCSGDI
>lcl|NZ_CP026520.1_prot_WP_042229132.1_1519 [locus_tag=PC41400_RS07890] [protein=hypothetical protein] [protein_id=WP_042229132.1] [location=1741291..1741539] [gbkey=CDS]
MSKQSQEVVDLFENVCELIESVRGSQERFEKRFEQFQKELTAQLNRIDDNNKVTAAEAKYDITFLAKEYGQLKLEMSRLN
QA
>lcl|NZ_CP026520.1_prot_WP_042229130.1_1520 [locus_tag=PC41400_RS07895] [protein=hypothetical protein] [protein_id=WP_042229130.1] [location=complement(1741585..1742037)] [gbkey=CDS]
MKRFFDQLQYFEDWRVYLLLLACSCGFGYLAYEQLMLFQLPESFLDKCLYIFRNPGPSFKVVGLSFLSAICFLALSIGSF
WNTYIAYENEKYYKWPVIFFIPIGIISAYLIYFFLSTFLQLLFLLLIIGILILLVLFNESKDTRTSRRRR
>lcl|NZ_CP026520.1_prot_WP_042229128.1_1521 [locus_tag=PC41400_RS07900] [protein=superinfection exclusion B family protein] [protein_id=WP_042229128.1] [location=1742168..1742671] [gbkey=CDS]
MRLDMGKILDILKLTPLYLFPIVLASAAMLFLPVDILKQLGIDPFTVKYKTWIGLTFIISFCFLITHLIYKFTTFVTGKN
GDRKLHKFRIKYLHGLTEEEKVILRYYLFHNTRSQKLDLFNGVVSGLIHTQVIYQASSVGDYYGFACNIQPWAWKYLKKN
THLLNPE
>lcl|NZ_CP026520.1_prot_WP_156972814.1_1522 [locus_tag=PC41400_RS29690] [protein=hypothetical protein] [protein_id=WP_156972814.1] [location=complement(1742944..1743105)] [gbkey=CDS]
MLNKLLSVGSVIFGVGLIGLNYSDSPDTGDAIALVVGAACVIAGIVGFFRADA
>lcl|NZ_CP026520.1_prot_WP_042229126.1_1523 [locus_tag=PC41400_RS07905] [protein=hypothetical protein] [protein_id=WP_042229126.1] [location=1743540..1743767] [gbkey=CDS]
MDRDELIRIVNYEILAVPEVLEELDITNQALSSLIERGKLVPVKEVGRTRLFLREDVEKRKKEAAELKGKYRPFD
>lcl|NZ_CP026520.1_prot_WP_042229124.1_1524 [locus_tag=PC41400_RS07910] [protein=hypothetical protein] [protein_id=WP_042229124.1] [location=complement(1744306..1744494)] [gbkey=CDS]
MEMFKTFLTLLGLFISYGVFRIANKKNKLELKKLKLEIRKLELEVAEKEKSAPIPPRNEKRS
>lcl|NZ_CP026520.1_prot_WP_042229122.1_1525 [locus_tag=PC41400_RS07915] [protein=RNA-guided endonuclease TnpB family protein] [protein_id=WP_042229122.1] [location=complement(1745562..1746647)] [gbkey=CDS]
MRIAFKFRLYPTKQQKEKIEFTLERCRLLYNRLLEERILAYKNEGRSLNYYDQAKTFTERKVYIPALKQVHSQVLQDAAN
RLDKAFQSFFRRVKSGETPGFPRFKPQKRYDSFTYPQAGYSIIGNKVRLSKIGDVKIKLHRQPQGKVKTCTITVKNGKYY
VCFSCETPAQKLPVSDKRVGIDLGIKHLAITSDGQIFDSPEYLRKSENKLKQLQRNVSNKIRGSNRRKKSVLLLAKLHEK
VANQRKDYAHKISRKLVNQYGYIAFEKLTKIKMLKNHNLAKSIADASWNQLVQFTTYKAESAGRVVVQVDPRNTSQECNN
CGQIIKKSLSVRIHNCSHCGYVEDRDINAAKNILQRTLAVS
>lcl|NZ_CP026520.1_prot_WP_042229120.1_1526 [locus_tag=PC41400_RS07920] [protein=hypothetical protein] [protein_id=WP_042229120.1] [location=1746935..1747549] [gbkey=CDS]
MSNLTIANRSSESLNYIIYLQNAILNTDIELNGTSHRKFPWVSIDLDKSVLVELQHVLINIWVDLVKNVRAEDAGTQLIN
LEILNRVTGDKVKAALEKLKASFLAWWNDPYIGGYNKLNYIDRNLLNKIYDRVKEKLNHSEATISLVIDVVFDVPVFDTI
MNSRNYVVIAIAELPDMRLFHDSGYIDERLILKLVEKIYNFLEI
>lcl|NZ_CP026520.1_prot_WP_042229118.1_1527 [locus_tag=PC41400_RS07925] [protein=HTH domain-containing protein] [protein_id=WP_042229118.1] [location=1747660..1748436] [gbkey=CDS]
MEKQKLISLLQPKDYERVYIHNEIFTELKNTDKIKSATNIAFAYCYYCLNCYLFRYCKYGNAYWFTKETLIEMLGFARNN
KTFDYLIKKNGLLDRLGLTETVSDLPVQSEFDDQFVSFIMYKALNDKEIFTLPHKYTVKKPIKAFRRYEDDVFDGTFYDI
SNTHLFSIHRFIQIISNPDLGAIGFFMFQYLLYNCAKFPSGYFITLIRLSEELSLSTGTIQKYINNLEKTGNIRIEQPKR
YLKEGEWKRYANTYFINH
>lcl|NZ_CP026520.1_prot_WP_156972812.1_1528 [locus_tag=PC41400_RS29695] [protein=hypothetical protein] [protein_id=WP_156972812.1] [location=1748563..1748733] [gbkey=CDS]
MSKLDTLVLIVVTHLVLAGIALILWACFNDPAVEDFLLYLLGLVWIIGGSVAIAKI
>lcl|NZ_CP026520.1_prot_WP_084706574.1_1529 [locus_tag=PC41400_RS07930] [protein=phage regulatory protein/antirepressor Ant] [protein_id=WP_084706574.1] [location=1749727..1750479] [gbkey=CDS]
MSKELTKFESEFGLVVNKEGKVVTNSLRVAAIYGKEHADVLKKIRKFIGLIPDLSRHGNFSESSYINEQGKSQPMFEMDR
QGFSMLVNKFTGDDATVFTYKYTEAFELMAEQLANPAMRYLALSEEDRAIAYFSEVKKKKELEHQLELSAPKIQKYDEFL
DADGYMTIGDAAKVLNIGRNKLFEFLRRIGILMNNNTPYQKYRNGTDKFKVIHTTRNREPISVTLLTPKGFDYIADKVKE
KFQTFNSKGE
>lcl|NZ_CP026520.1_prot_WP_042229117.1_1530 [locus_tag=PC41400_RS07935] [protein=hypothetical protein] [protein_id=WP_042229117.1] [location=1750489..1750713] [gbkey=CDS]
MQGKQDLSSLEELLSGSIDQLDRLLINIMTTVKENGFESDVFEKASSDNEISIDTLEYLIHVNKYIDEREEEYE
>lcl|NZ_CP026520.1_prot_WP_042229115.1_1531 [locus_tag=PC41400_RS07940] [protein=HU family DNA-binding protein] [protein_id=WP_042229115.1] [location=1750706..1750984] [gbkey=CDS]
MNKEQLVKRVAKELDMSNKHVESVVDTVLETITVTVGDGEEVALHGFGKFEIRSRAARKGFNPSTREEMQIDESKCPGFT
AGKNFKEAVKSK
>lcl|NZ_CP026520.1_prot_WP_042229114.1_1532 [locus_tag=PC41400_RS07945] [protein=hypothetical protein] [protein_id=WP_042229114.1] [location=1750996..1752204] [gbkey=CDS]
MNHSNLTRGSEESLDRFLLRLGDDKEIYGLDNWHQIRDVMNAESNEDFSESKWRKDYHLLKRGYDLHQAQLNPDGIVLLQ
EEKKLEAQKEYKKIQTIKSELNKNLNKAARREVMWDMIKDSIEKIPVPKFQPIIYPPTRDKVGVLSFADIHFGKYFKSLK
NEYNEDIARDRMSLLMGETLEIIRQHGFEHIHIINGADSIEGMTLRTSQLQSIQSGLIDQTIKFSKFIASWLNVLSQYVR
VTYHHVPSANHSEIRPFNSNRGEYPSEDLELVIMNYVHDVLANNPRIEAPLYDKDYVKLDVLGYNVWALHGHQLKGKKNA
IRDLSLLHREFIDYLYIAHFHHAGTLTVGEGLTNDIEIIQIPSVMGADEYSDSLMTGAKAGANFSVYERNKGRSISYSLK
LN
>lcl|NZ_CP026520.1_prot_WP_042229113.1_1533 [locus_tag=PC41400_RS07950] [protein=hypothetical protein] [protein_id=WP_042229113.1] [location=1752356..1753243] [gbkey=CDS]
MAEKRKPSARRPAAEKKKCEKCKQDKNRDREFYNSNSKCSDGKASICKECFEVLVEEYGFEFFIQFLQHVNRPYLSDEWI
KWEENCGKYLKNVSSLPQYKDLSYSSSIEIPKSIVQKTETDIDQLLMEKWGEGYEPEEYRAFERKYATLSNNYQERTAMH
TEALITYIRYRVKEELATARGDAREAKDWGALAKDAATAAKINPSQLSKSDLSDGLDGFGQLVRSVEQSVDIIEILPKFK
ERPQDSVDFNIWCYVNYVRDMKGLPSAEYSEIYQFYEERKKEYESMKQEADAYEK
>lcl|NZ_CP026520.1_prot_WP_053228693.1_1534 [locus_tag=PC41400_RS30070] [protein=hypothetical protein] [protein_id=WP_053228693.1] [location=1753233..1754867] [gbkey=CDS]
MKNNSCLQADGFSKNLDKYYDLVSWARWNPDLWWDLITPPTGGIRLDLDQRVYLRCIARFVSNYMVFPRGYGKTMLEVMG
MVHTAIFFPDIEISMTAQTKENAAKLVEEKFREITKFYPMLLHEIQGRPLFSKDSVEIVFASGGRIDVMANAQSSKGARR
KRLNAEEAALLNNALFEDVLEPIVNVPRRTIGKEALINPEELNGQINFMTTSGFRGSDEFERNLRMIDEMAELKGKIVLG
SDWQLAVKYGRGEPKSAILDKKSKLSPTFFAMNYESKWVGASDGALVNINKVLDLRTLSQPEFEGDGKSSYILGVDVARS
QSKNNNQSSVAVLKIKRNKEEKITRISLVNLINLPIGMNFTGQAIEVKRLQNLYNAKTVVVDVNGVGTGLCDELLKDTVD
PNTGESLGCWDTINTDHEPEIQRSDKVIYALTAQGINHDIIVSFIDMVESGKLQLLVKNVDNSYDINDTDSTKKSLPHIQ
TDLLIEEIANLKLRQTQSARYTVEQLTKRVDKDRYSALAMGLWYIKNFEDKQRKVKNKMVFLYN
>lcl|NZ_CP026520.1_prot_WP_042229111.1_1535 [locus_tag=PC41400_RS07960] [protein=hypothetical protein] [protein_id=WP_042229111.1] [location=1754885..1756384] [gbkey=CDS]
MSDTVNIVKNEKGKVITAQLAYTPYNSSNHYGSYKQDAKYSSQDIKRLLSNHEKNSVALREMSDYLYITNGTYRRLIDTT
KNIISFEYLIEPSVANIEKLNTKSYQKKHAEVKVYLADCIMDTTVDDILFKTIKYGRYSAYDRETFLQPLPLDYTRIIGI
ASDGNPVIQFNFEYFEQFTDIREREMQLNGYDPLFKKMYLDYKNRIDDPKVNFSNEYNWRTLPHEKTYTIKIGSNLESSE
GIGLLYGSIDDIQYYDELKELDRGVVNSQRRKIIVQQIPVDKEGNSILGEEEIIQSHENLKALLPPNVGCLTVLGGTKFD
DIPLQLSALEKGKVQEVQSDVLMNAGIGEGALKGGNFSAGVLNVEIITNSLVKILGQIQSIWFNRKFKKFGNSQFQFRMK
FLGITSFNRDKVIAAFDGLLDKGGSLSHSIAARGMDVSNYIDIVAIEDVLEYKDKFKPLQTSHTISGKDGGRPPSSDLSN
DNTVKDKTNGGNMAPRTGQ
>lcl|NZ_CP026520.1_prot_WP_042229109.1_1536 [locus_tag=PC41400_RS07965] [protein=hypothetical protein] [protein_id=WP_042229109.1] [location=1756399..1757409] [gbkey=CDS]
MQKQLVLNSLKTVLSEENSNPLFLEMDILLMRNAVNLNNYYFTESFIDEIVQNQNKYNGLPLVVDLERLLENDYDGLGHQ
YSEFLNKFFTQQIGNYVSFEKKYDENGIAELHGTARISKRNEDLVKVINELYVTSGLNFSVEVIAQEYRVDENNWLIIDA
HPGNLLVSDCLVSVPAEPSSKALKLVAQLTKGGKEKVSLEEMKIEKEKLEAQVATLTAQLNEANQAVTTLGKEKEMLTAQ
VDDLSKYKNDTLEKAEVDKINALIAEVKESLSEQELTLVNEVAEAKEFTKVQFKVNEICAQKYKESIRLNNSTGGSFFSV
TKQKDLGNDELTKFNM
>lcl|NZ_CP026520.1_prot_WP_042229107.1_1537 [locus_tag=PC41400_RS07970] [protein=hypothetical protein] [protein_id=WP_042229107.1] [location=1757464..1757943] [gbkey=CDS]
MANKYGIVRLDRVSGTPDGRLKAKVDLENGQFVVQDEVNGELKLPAAVTDRVVLVASVCHQYESKNEGDFINKAGSNLKP
RTYEKVAGDIFTVTAIGYTGGTSPRGDFNSIQVGDKGHVQPSGKLVFYAAADTTAYLTVEVIGKTKLNGDNAVQVKIVK
>lcl|NZ_CP026520.1_prot_WP_042229105.1_1538 [locus_tag=PC41400_RS07975] [protein=hypothetical protein] [protein_id=WP_042229105.1] [location=1757972..1758943] [gbkey=CDS]
MIEQRILNCAVDAALGRSTDPKFTPEQRVEGMRAYLAELGKDYRRNEILINEVIEQTVDAILPIKITERLGVFADVSQVQ
DGVTKKWHVKNGKITAAYTALGVEQPRQKLYKGTFTTQTAPISGAVYAEYEDLITGRVDFADMINQLVGAIMDQIYAGIQ
DALIGAFGTTNNANRFVGGSFMQAEFDKLKGTVQAYGKPVIFGTAVGLTGISNGAGFDWNKTSEADRLDIRNLGHVGKYK
GSDVVELPNSFTDETNAQKLLDDGYIYMAPVNADKPVKIVLEGSMHIRDKQERDWSTTKEYYRKAGISVLAINHMALYEN
SSL
>lcl|NZ_CP026520.1_prot_WP_042229104.1_1539 [locus_tag=PC41400_RS07980] [protein=hypothetical protein] [protein_id=WP_042229104.1] [location=1758992..1759486] [gbkey=CDS]
MTKTKMIRVVNNMPTGSLGIVLTDGKPIVLPKPGNYKEVSFDNVLNIFNSCKLIQRGHLYIDNKDVRIEFGLEDGDQVDI
NALSREELGELVKESDAKQLEAILAADTSDGTKEKIVLLAREQYNNESMDAKKVKVIETATGMSVTEEAPSESETTKSSM
ISKK
>lcl|NZ_CP026520.1_prot_WP_042229102.1_1540 [locus_tag=PC41400_RS07985] [protein=hypothetical protein] [protein_id=WP_042229102.1] [location=1759499..1759930] [gbkey=CDS]
MSVTLEKIYDVFYSSINEDLAKHPELDVNVPDNIVDTFLDSAAYEYEENFGEEIVYDSNKTEILNDIPRHHIRLLGKLIY
KSYMERELNSCLRLANHFNKRSELQVTGLQAKIVALKEITGRLTRETNRMFTRGILKGVSKLG
>lcl|NZ_CP026520.1_prot_WP_042229101.1_1541 [locus_tag=PC41400_RS07990] [protein=hypothetical protein] [protein_id=WP_042229101.1] [location=1759935..1760579] [gbkey=CDS]
MKDYSKWMSPNSDSEEILFDLCEKSKVKINDFTAGAVSYEAVINDANEWQKDVDLIILIKKSEYNIHCGDYIQANNQCFL
VLFSPEDRNFFWSVKVRKCHSLIRWLNETGSIKEIPFLLDNVINSLGIEMGRVVNLPDEERRILIQRNSDTLKFAKSKRF
IFDNRGWKITAIDGLSRENIIVITLLEDTINTYIDNVDECIANFKNEKTSKPGI
>lcl|NZ_CP026520.1_prot_WP_042229100.1_1542 [locus_tag=PC41400_RS07995] [protein=hypothetical protein] [protein_id=WP_042229100.1] [location=1761080..1761799] [gbkey=CDS]
MESVKYILGKPEVIQEAGEIFPIKVKDYDEFNDCRNVLYINKNHFSNEYQEYPLLDLIVYGYKDVEMIRQLKLLIKLVTG
YDPQLHEDKNGCSFIISGDKKVYSGNYDSFRRIVMRQNLLFEQKVYKNKLTQEWANKAIEAKSKNGIPITFEDMITTVSV
ITGKTYDQIGEQTIYQLTADFKRIARDKSYHTSIAMKCAGAEKVSIEHFAEEINMFENPYDNLFVGKDNLNNLNKALKK
>lcl|NZ_CP026520.1_prot_WP_042229099.1_1543 [locus_tag=PC41400_RS08000] [protein=hypothetical protein] [protein_id=WP_042229099.1] [location=1761842..1762576] [gbkey=CDS]
MALIADVFIALLVDPSGQTFASTTLQSGDIDFNVKENEVRGGRGNALQGVLHSDRDISVKMTDSLFRYDWMARQLGQTIV
TGAGKAYAMPEWYTASGTTTIKISLPEAPIADTLVIYKADGTLVSKANYTLSNKDVTFTTGVTAGEKVDVRTYQYATSAK
TETINIDNSVFAKGTKLILETLEIDNDETPLYKIQYIFESAVPTGNFSVKTASERNGQAQQFDLKVLKPKDGNVIGQIVR
IPLT
>lcl|NZ_CP026520.1_prot_WP_042229098.1_1544 [locus_tag=PC41400_RS08005] [protein=hypothetical protein] [protein_id=WP_042229098.1] [location=1762690..1763220] [gbkey=CDS]
MTKAKNKKLELTLEQVTVQDEQYDNLKKVVFDDGAYTEVYEKFRPSRIETMIEQLANFLEEYEKHQGKIEDDSTIAKFLY
LHVVVHFSTIVSEMPSEFSTKLDLFTKLADNVYVRKLEIHFDKSEVEKVYEMIFKRLDDYQNLIEQNTELSKKMSERVDK
LNIQNKELLNWNQENA
>lcl|NZ_CP026520.1_prot_WP_042229097.1_1545 [locus_tag=PC41400_RS08010] [protein=hypothetical protein] [protein_id=WP_042229097.1] [location=1763213..1763644] [gbkey=CDS]
MPDFNSIKELQRYIQTKANLALKNEVATNTVEAMMKKIDEVVYDVYEPKVYEREKDHGGLTDPNNIRVQMINDDTVSIEN
IRSDGNRNVVEIVETGQGYYYSFDYTNKPRAFTGATRQELKTSKSHIKAMKLGLERQGIRTEQ
>lcl|NZ_CP026520.1_prot_WP_042229095.1_1546 [locus_tag=PC41400_RS08015] [protein=phage tail tape measure protein] [protein_id=WP_042229095.1] [location=1763703..1769285] [gbkey=CDS]
MADDLRILITAGLNVGKTTTEINKAVAALDKKIDKLQLRIDVDQNFTKNIQIFTGAIEKLKKQLEELNKTFQNTNAVNNF
RRSLSDTDNQLRTLGNTAQASRTHVTGIGEAMASAIAGTLTWGVATTAIYGSKRAMEEMLQVIVEVDTQMTHLKRVMDAD
TNFEEMLNGSIAMANELGQKITKINEALIDAAQAGYAAQEALDITRTATLASNVSDLSPKEAMSNLIAAMKAYKIEAKDT
IQIVDKLNQVDNDYSITTKTLSQSMEKAGATAATYGVSLDQLIGYTAAIGEVTRESGNIIGNAQKSIFSSLYSKKAATAL
AEVGVAITDVNGQNRKASEIINELGSKWKSLSMTQQQHTALSIAGKYQLTRFLALMGNFDIAIKATQTSLTSQGSAFREN
EKYMDSLQAKLNLNSAAWQNLAITIGNSGVKSAFSAIVSSITGMVKGFTSLTEATHGLNIYLPLAAAGVYGLVKAFNALK
AAGIGLKSAFGWVGIALIGVEMLISAFMGGTKSVEANSQALAENANQTRMNADNIQSLVDQYNKLEPLATGNSEKQKELQ
SVLNQINSIAPQLIEHTGKYGDSLSLNKEKANEYIESLRQMTKEQVEQAKLANGIELNNVNTEIAKAEDKIKKFGTNVSK
EFSDLQVYQKLFDARTAEQAAINFTNTVKKMKSEIKTLYESGDVSKAIVKEGELKNLQKQYSDFKRLLETSGASLKEYAE
VVNDKNGLEQKKTQFVERERALDKLTQTQDKAKGSTDQLSGAFNEQTGEIENNTDALQDNNDQVATGIAKNQNFIKQFNT
TTNELKTLNNAIHDVTKGQALSSEATQDLILQYPELASYVQISAEGFTIEKSALDKVRQAKIDKAKADLENEKASTAAAI
SNTIARMQAYGKEIGAIRSVAEAKGKLEDVNRKIIENSKPGIEEELGFKVATIPNQYREKFEQQKKAQLAEANATRDALR
GFISEMTGFDAQIAVLQKTLDNPNLGVSSSKTKTPKEKNAKTKSEPSAYDETEKYKINEYERALRSLDDELRESEAIINR
YPDTSKDYRDELTKQVDIRKKMQALAHNEAERLRSEISDVKNKIAVGKLNQKQENDLLKQLEQKENNLSQLSNNWWDYQV
AVESTQKKITESIAKYNKEQFDISKNWIDQEKYYKRLSLEDELAAWERVQARYEQGTEERLNADKEVYRVRQELLSKYEE
DFNKVMEKASQAADETIEKLRKKMETKESFNPGELADSVDAIMASIDMIDKKYPNGYSPIDTSDEARTRLNAYKGNITGL
AANMKSIMEMPYQNADQLKNMIDGQIQFANNIKKIMNETNALIKQTETAYRMQEASLENQIKLHQQRNDKIIEGLNKALE
VKDTFNSTDFNTSINSILSALDALDGKYVKGNFVESTLGTRTDLADYKKKVTEIAEAIKNYQNNANFSSSSITEQTKYIN
GLKTQIDLLNDSIRARELQYKNEEEALAQLIKTKEKSYDDEINKQKEALKNLDEQIDKEDRLKKLKDINDELDKVKNDKR
FSYIGADGKEILTYDKGKVAELEKQRDELLKQYEREDVKKALQDSIDQLEKTKNETIQKLQEQLDKTKGIHQNELESQKL
YVKSLEDLYRIGLQDVQNKIGQLKSHYDTEIANEQARLDTLRQVHQSELDSMNMYAGYLSQLNQFLTQDTQTKIDALKTI
YDKEAEDLKKHLKEMVENTKNGSIEFHKVLDEYKNGAISRLDSMVSAAQDRLRQMQAIMAAMASMSGGAAPSSAIVPKYH
TGGVVGSAPLASNEVPAILKLGEEVFTQDHIAKLKSIIMQPINFVGSILGNVRNTIPDTPIRNNQAIAGPTYNLSNFTIQ
ANNPTELFSGLNNLIKSNRT
>lcl|NZ_CP026520.1_prot_WP_042229094.1_1547 [locus_tag=PC41400_RS08020] [protein=hypothetical protein] [protein_id=WP_042229094.1] [location=1769332..1770186] [gbkey=CDS]
MTIIGYDFYGTQFFHSPGNAIKDIFYGELKGGIYDELEIRDRTDMPTDKNKTKWEIDHRAIFKFVGDLEGGNISNSGLEI
VSFAIKRRRVNEIDSVTLDHIPFVNNLSINYEDYTQPNDELIYTIVPIAQNDLEGKPVEVHVKSDFVGWWLVDKETDNTL
GFDSAIGEIGSVETALNQGRTVIETLSRYPSVYYFDKDYHTFSLSTAIIPSEFERSGQNYEKIVNTFIRDHRPFIVKSDT
GRIFVCDCSNPSFSTPLNTWRDYDYGVLTIDFLEIQDYNEFMNE
>lcl|NZ_CP026520.1_prot_WP_042229092.1_1548 [locus_tag=PC41400_RS08025] [protein=hypothetical protein] [protein_id=WP_042229092.1] [location=1770198..1771376] [gbkey=CDS]
MQYCSSDYLAALKSPIKQMYLKFEFYDAQMNYISEFTKQITAQDSGSISVDASRPVRRSFSFKLMNRKNEFNFGEDNLIW
IDKRVKVFTGLKLRTGEIEYIPQGVFILTEPEDSHTVEGKITSLSGMDKMFLCTDKRGKFRTETTIDAGGNIGAIIRVLA
QKYGETQFNFDPITETIPYTFTFQPSDNIYKAMSDLALLAKCELFYDVNGYLRLRKIDLNQFEQSPSVWMYVHGDPNERF
YAGNVRKFDENQLANRIMVLGGSSQIAESRFELIITETNPLWKGHPYSIEKIGEITYEHNSGNPDSLLATNEECKWRAKY
ELMNRLGYAERVSLSLSPNHLHDANDVIEIEDVENSVTGKYLLQSFSLPLTPQLMNCECMKYRRVIADWNFI
>lcl|NZ_CP026520.1_prot_WP_042229090.1_1549 [locus_tag=PC41400_RS08030] [protein=hypothetical protein] [protein_id=WP_042229090.1] [location=1771389..1772732] [gbkey=CDS]
MTNEDFIKAVRQIVNIELKENNLAIGHWHLGTVMKVKNKTLLSISINGSATSQDVPCNPDVNFIKGDLVYVVYVNGDSKN
KFVPYKRFADENYTGNSVANQVLGNLSLNHKEESGLTAPYNFTVIQGTRGGVQYSELWILVNAYYDYKTKRFKRKNVNNY
SFGWQFQGMGTYPGEENFGDYSNQGVNLWKANGKSAYGVNDPARDMTNEDIGAVQSDGSWREFGIMLGWNNVFMCDSYGG
MTIGGAGFEIDGNGTSPFKRVSLNKFNGGSVDSSKEYEQYGYAYNGNLWNAFHGMHGSDQNNRNSFFWGLVSPIDFYDQG
YHNPYSNQASMEEVKFVLRMKPKNVGHQAENWKDIMSMDMEGNMKLKERTVSATLVANATVNGTDFNFNYPDSTWNKSNT
FIAGVIGVQSNGSLKQFGAINATFTDFGAYGYLGEGFVSAKILISKY
>lcl|NZ_CP026520.1_prot_WP_042229089.1_1550 [locus_tag=PC41400_RS08035] [protein=hypothetical protein] [protein_id=WP_042229089.1] [location=1772751..1773758] [gbkey=CDS]
MAIQKPYNISLKGTTVDAKDSIKVSWQSSGDMSTGFQVQIYRNQDNNLIYDSGRITSFAKTYSLPAFALPNGYELKVKVT
IWDSGNNFAVSDTEIFQTSSRPNVSVSPIGIVNSQSYEFSATYSQNEDIPLRSWISFLYDKDKTLISQSNVRMSTPLVYL
INYLKSGSSYFVEFQATSNKGSVGTSGLIPFSVSYLQPDISINLDAKNIKNAGIQLSWKTIQIIGSTTKQPVYLNGEKLD
LRDDVFTFTDGFHVDADFTLKLWFEDIKSYTDLLVLKGQNGIMTLQYWDDDQRFHLFREVYNYRSHYASQPLTGNGYFLC
IQQIGGDMNIFGEVN
>lcl|NZ_CP026520.1_prot_WP_042229088.1_1551 [locus_tag=PC41400_RS08040] [protein=hypothetical protein] [protein_id=WP_042229088.1] [location=1773763..1774380] [gbkey=CDS]
MAVNLSTFPLAIDQFQKHYEIQAQDIPGIQRFQELKLKASRTPSEETELANLTVQYREKFISADDFNRFQDALVNMEIFI
KDNVEGYILTKQGEFTNYVDAAKNTVQTTKDSALQTIEQKKENVITYLDGTTAGALRNDIGVMGDLETVNKQSLVKVANE
LRSDSIRVDNKVGVLSTLNTTDQSNIVKAMNEIKSNLEIAYWMGV
>lcl|NZ_CP026520.1_prot_WP_084706573.1_1552 [locus_tag=PC41400_RS08045] [protein=hypothetical protein] [protein_id=WP_084706573.1] [location=1774382..1774711] [gbkey=CDS]
MINTPKRLYVNQPGMTVSTAYTVPALTTTIVKNIHIANNTAAVSTVTIHLVPNGQTATNANKIIASYPVKPNDSISVDLS
SVLHSGDTIQVSQGTANAISVYISGVECV
>lcl|NZ_CP026520.1_prot_WP_129112459.1_1553 [locus_tag=PC41400_RS08050] [protein=hypothetical protein] [protein_id=WP_129112459.1] [location=1774708..1776297] [gbkey=CDS]
MSIRRLGDMRIKPGASGDVQPSDVRAGKTFSSEIDTDLVGTLPDRGAMTFTPSGTSTVSIPDGIHASSKITQVNVPADKV
LTGTTIASVAGTMSNRGAPTFNVSPNNQLISPGYYSGGIVAGDANIKPENIRQDVWIGGVKGTGLKYGVGEWIPFPSSKP
YRLLQKFTYTIPFTDGAKNGQINIAGRKGSFLAWYYYWWSNPNYTTNYRSFIVCLNENGTERWKNSLGTNAPSHSGVYVS
KDGLYGFLTKVDQISGESSTYYNLTVIKYRMSDGAVIWQAPTNYISFSPTGFFEDTNGELFIYGSYLGSGDGSNTIVRYN
MASGTSSGFISGGSGRTESIAITPDGRYIYVKSTGYSDYNKLQGTGTTSNNEFGKRLFSTYGNYYFSRTNKILYVPSTGK
LIAIGDSNAVIGSGPLPSGLNGTEITMMLEMNPDTYYPTEGFYRGLPGGYWAGYATLDDTGYALCGLSSSKGVYLLDGNT
KTTPMLIPNSFFTSGYAPSMASWDGSNYLACADSGYNSATVYYLNAYYI
>lcl|NZ_CP026520.1_prot_WP_042229087.1_1554 [locus_tag=PC41400_RS08055] [protein=hypothetical protein] [protein_id=WP_042229087.1] [location=1776310..1776594] [gbkey=CDS]
MSKAYILLNENGDLTSTFFEKEFAPKEAIEVNAPMLDQDKMNTHYSFLTYDKETKVLSYRYEEIYKGPTLEQQVEELKAQ
NAQMLLALTENGLL
>lcl|NZ_CP026520.1_prot_WP_084706572.1_1555 [locus_tag=PC41400_RS08060] [protein=XkdX family protein] [protein_id=WP_084706572.1] [location=1776608..1776745] [gbkey=CDS]
MDWFKTVKTYYDAGYYTNDNVKVFVIKNKITVAQYKQITDEDYSI
>lcl|NZ_CP026520.1_prot_WP_042229086.1_1556 [locus_tag=PC41400_RS08065] [protein=phage holin family protein] [protein_id=WP_042229086.1] [location=1776819..1777217] [gbkey=CDS]
MNESILKLIFSTGATITTYAFGGWSAVLGVLVALVVIDQVTGIVASIKDGTGLSSRVGFIGISKKVFIFVVIAMCHWFDG
IMNLNGILMDAAIYFYIANETISIIENLGRIDVLIPEQLKQIITVLRNKNQK
>lcl|NZ_CP026520.1_prot_WP_042229084.1_1557 [locus_tag=PC41400_RS08070] [protein=M15 family metallopeptidase] [protein_id=WP_042229084.1] [location=1777229..1777945] [gbkey=CDS]
MGLQLEFILSKSSKRLSDLHPTVKIAAEKLIINSYKRGVPIVITQGLRTFAEQNALYAQGRFGDKKPVVTNAKAGESFHN
YGVAIDFALLLPDGKNVSWDMQRDGDGDRQCDWLEVVEEAKKLGFLWGGDWINFKDSPHFEMTFGLTIKDFQSGKRPPIV
KESIKEDEDNMPLPLNDYEWKMLNGIWSQRYNKGEITDWKWIDKIRCKQLNAGELAFLNAVIAAKKDCLSVEAESAGM
>lcl|NZ_CP026520.1_prot_WP_053228691.1_1558 [locus_tag=PC41400_RS08075] [protein=hypothetical protein] [protein_id=WP_053228691.1] [location=1777950..1778570] [gbkey=CDS]
MKKINEGKKFEKNYESAVNRTNYFFLRLKDSAKWLRGSGATFTPENPCDGIQYAAPFLWLVELKSTKGSSISFNPERPEE
KPQNKTTQVMIKAHQVKSLMKFATKDGLIPGFIFNFRERKTKTKSYPDETFFVHINDFISWSKSSGKSGINRDDCKTIGV
KIHSSRTQKYDISKFVGEAVSAFIQRGYIKSEFIKGMSEYLLTISK
>lcl|NZ_CP026520.1_prot_WP_084706571.1_1559 [locus_tag=PC41400_RS08080] [protein=phage antirepressor KilAC domain-containing protein] [protein_id=WP_084706571.1] [location=1778751..1779533] [gbkey=CDS]
MTEKTNVREIEFHGTKLLCVQSEDKKIWVGVSFICNGIGLSEGQRDRQVKNLQDDVLLKRGCKKLPVKFDGQVRETLIID
IDYLPLWLAKISITPSMLSRNPELVTKLIEYQLKAKDVLAQAFIYKKTEYHIPDTYAEALRLAAEVQFQLEIAQPKAEKY
DEFLNSDGLIQLNEVAKHLKIGLKTFYAILRTEGILMKQKKPNVHIPTAKYANKGYFSIVREQVPNKDSKMVWHSVAYAT
HSGFDFLNKFISAYIQEETA
>lcl|NZ_CP026520.1_prot_WP_042229083.1_1560 [locus_tag=PC41400_RS08085] [protein=hypothetical protein] [protein_id=WP_042229083.1] [location=1779556..1779768] [gbkey=CDS]
MRIEVSDKIVAVWLNESDLHSEELKQLALECRRDKKKLVKFVSGKNDIAETVEKLIMNNLNIGPNKSRYK
>lcl|NZ_CP026520.1_prot_WP_009671540.1_1561 [locus_tag=PC41400_RS08090] [protein=hypothetical protein] [protein_id=WP_009671540.1] [location=1779862..1780044] [gbkey=CDS]
MIRKNEKKVILGIKDIMERLQIGRNMAARILASDELHVFRINEKRFVHEDVFEKWLKGDQ
>lcl|NZ_CP026520.1_prot_WP_042229080.1_1562 [locus_tag=PC41400_RS08095] [protein=tyrosine-type recombinase/integrase] [protein_id=WP_042229080.1] [location=1780131..1781243] [gbkey=CDS]
MPEPKKNPKTETWEFVFDYYVKGVRKQVRRRGFQTRKQASKAMIALQNEVLEEKFVCTDKTTVGAFVQNWIDTVRKLEID
ETTMYNNKLYLKNHIMPRIGKLRLQKLDHIVCQEFVNEMHEKGYARNTIDRVVTMLKKALDKAVQYKLIRENYMRQVTLP
RVVKKAKEIWTAKQAIQFLKATEDRRFHCVYALALLTGMRQGEILGLRWRDIDFERRIISINQTLTHYGKTIKSGAKTAA
GIRTIAMPNLLITILKKQYEKYDELKQKLGEKFIDMDLVIFNLSNGKTVFPSNLTKVYTADVKRAGLPHITFHSMRHTHA
TMLIEQNVNVKLISERLGHSKIGVTLDTYSHVIPSMQQEVADTIDEIMVM
>lcl|NZ_CP026520.1_prot_PC41400_RS08100_1563 [locus_tag=PC41400_RS08100] [protein=YunC family protein] [pseudo=true] [partial=5'] [location=<1781309..1781509] [gbkey=CDS]
YIHRGALDVALLNERLSDRQIIAARATGVKTIGELLDAPLESVTHTAEEIGIFAGMTGRDAIRKMM
>lcl|NZ_CP026520.1_prot_WP_042229079.1_1564 [locus_tag=PC41400_RS08105] [protein=hypothetical protein] [protein_id=WP_042229079.1] [location=1781808..1783109] [gbkey=CDS]
MTINQFAEISGINSGTLSIILHGNRPIAMQQLDRMTAGMGLHEGHFYDVYIEECFFHSNPDWRRLGPLIHRCAELGKLDL
LLRVVRLMLENLSYLPLLFETAEQFHSEGRREAAVLLYECIAESEKYQHSERLALCQYRLFTLAIGDDLKANLRAATLFE
SYVPRLDVSDQLDALNQLAYVFGTLHEWEKVDTLAKELHRIATIQYRLEPASEPGRETIKQPKRPLCYYILQALLIRSVV
CEERGKFDQALYYVSLYAKPDWVREQSDEERRIVEQFREWAVANTYLYRIMSGDLVVLPEYINCIASRKDDIFTALCKIV
QIANRYGYEIDDILDRFSSHLPYQTYQSEFGEYNKPVMKERYAQFLSDLAAYHLTQNRRKGIKFVLHGLEFSVRINSERN
IIQCMALFERFRHLAGVEEKNEFKHLVNEVRLG
>lcl|NZ_CP026520.1_prot_WP_042229076.1_1565 [locus_tag=PC41400_RS08110] [protein=aspartyl-phosphate phosphatase Spo0E family protein] [protein_id=WP_042229076.1] [location=1783439..1783663] [gbkey=CDS]
MDDIHQHIEEIRREMMRLAGEYGFSDSRVLTKSQELDRVLNTYQREKCKLKEQRYPGVIHLSLSSVTGSAEFIN
>lcl|NZ_CP026520.1_prot_WP_042229075.1_1566 [locus_tag=PC41400_RS08115] [protein=hypothetical protein] [protein_id=WP_042229075.1] [location=1783676..1783876] [gbkey=CDS]
MPNSIIQLFNSDQLLIVMFLMWVLISGMFMVLHLDGKKSKRTRDEVDPKREELPEHEHRKNERGRL
>lcl|NZ_CP026520.1_prot_WP_042229073.1_1567 [locus_tag=PC41400_RS08120] [protein=glycosyl hydrolase family 18 protein] [protein_id=WP_042229073.1] [location=complement(1784051..1785733)] [gbkey=CDS]
MTSYTTPRKKMKMIHLALAFTLGLGVVPALASSQANAAGTAAAACQDPAWNSTTAYTAGKRVSYSGKTFEAKWWTQGEQP
NTANTWGAWKYVSDCGSTGGGTTDTQAPSVPSGLKATASTSTSVSLSWVASTDNVGVSGYDVYQGTALAASVTGTTALVT
GLTASTAYSFTVKAKDAAGNVSAASAPVSVTTPAGGTTPSPDRRYVAYASTWNTSIYDLKLENIPNYVTDINLSFARPDT
KYVKGSYAFDQEVAGFEFVEGASTGRGQLKFTPQQSQDLRSNIAALNARGTDTWLSVGGWSYSQGTQWGNFNAPHVVDLA
LDLGASGVDIDWESSGSNCNKAAAESFSCSKDAEIANIINSLYNEIKARGAKLKISIAGWSTGAYYVKGSPFEEGKVQYG
SPFGGTMYRVVKDHGSKIDMINLMSYDGGVYYDPREGYESYRAIYGGTINMGLQIAPEGSGGAVLKLNAAPGTVYDAEML
NGQNNVASAYYNVETLVKYVKNKGKSFDGFMLWQLWKQRVHLPVPSDGATENSAGQYVCRNLPLAGDCSQTIPTLPKLTP
>lcl|NZ_CP026520.1_prot_WP_241688817.1_1568 [locus_tag=PC41400_RS08125] [protein=hypothetical protein] [protein_id=WP_241688817.1] [location=1786027..1786257] [gbkey=CDS]
MTRGKIIYRMTELLFLAGLIYLIRGYMSVYLVRSVGLSDWQAAQVVWIILLVCLLVCAAIGILQGLQAYLKKKGKF
>lcl|NZ_CP026520.1_prot_WP_042229072.1_1569 [locus_tag=PC41400_RS08130] [protein=aldo/keto reductase] [protein_id=WP_042229072.1] [location=complement(1786542..1787519)] [gbkey=CDS]
MNILPLKKRGLNTSRLVLGCMPFGGGWNSDPVTKEDETQAQKAVEAALSIGISMFDHADIYTRGKAEETFGRILKNKPGL
REEMVIQSKCGIRFAEGDVPGRYDFSKAYIGEAVDGILRRLGTEYLDVLLLHRPDPLMEPEEVAEAFGRLKAAGKVRYFG
VSNMSAAQIRFLQRALPDQLAVNQLEMSLARHDWLDQGVHVNQKAGTSVNFSEGLLEYCQMENIQIQAWGPLAQGKFSGN
LGESEPEHIRKTAELVGKLADEKETTREAIVLGWLMRHPAMIQPVIGTANAERIAACKDAERQAELMTREEWYSLYVSAR
GSFLP
>lcl|NZ_CP026520.1_prot_WP_042229071.1_1570 [locus_tag=PC41400_RS08135] [protein=DinB family protein] [protein_id=WP_042229071.1] [location=1787767..1788255] [gbkey=CDS]
MFVSLNEFQQEWKQEASLTGKILNTLTDESLDQSVALDHFSLGELGWHLSQAIYGLFSAAGLKFEGPERGAGVPESAKEI
ADVYEQISSNLLRAAGEQWTDETLKQSITVFGRETTIHGIMRFVIQHQAHHRGQMTVLMRQAGLVVPGVYGPSKEEREQI
GK
>lcl|NZ_CP026520.1_prot_WP_241688818.1_1571 [locus_tag=PC41400_RS30205] [protein=hypothetical protein] [protein_id=WP_241688818.1] [location=1788766..1789083] [gbkey=CDS]
MRIQKVTEPVNGGRRLLTWRYYSLLLLLTVGLEYGALVAVHLPFGDEMTGFLVMMALFAFSVLLAIFTFVFGLLTCLENR
QPMLIVPCALLTAGLYLAIIRWEGM
>lcl|NZ_CP026520.1_prot_WP_042229069.1_1572 [locus_tag=PC41400_RS08145] [protein=GNAT family protein] [protein_id=WP_042229069.1] [location=1789090..1789626] [gbkey=CDS]
MIRFAYFGEEDFDQLIEWSGDEAFLLQWAGPQFKYPLTSSQLEAYMEGANDPEQSGRLVYKVIDSETDLTIGHLSIENID
RHNRSARIGKVLVGHTARRGRGTGQAMIREALRIGFEELRLHKMTLGVFDFNEAAIRCYEKEGFVREGLIRDSKRMGDTY
WNLIEMGILESEWHGSSR
>lcl|NZ_CP026520.1_prot_WP_042229068.1_1573 [locus_tag=PC41400_RS08150] [protein=inositol monophosphatase family protein] [protein_id=WP_042229068.1] [location=1789702..1790499] [gbkey=CDS]
MEDILRAAKKVAIQAALQAGQEAKNRFENINSILEKDDFGDVVTEVDHLAEEIILTHIAKSFPDHCVDSEEIGHNGRQSD
WMWIVDPLDGTNNYAIGLPLFATSITLMHQGNPVLAVIYEPMVDRLFVSAADKGAFCNNIRVQTKPCPSIQRGTIGWIQG
HKVQNEARAVKLRHHLDVRFKRMMRVWAPTLQWCMLAKGDLDGIVLYNSEGTDLYSGILMVQEAGGLVMDYEGKPFRGMS
PEPYLIACHPAHKEHFLKTVAEGMG
>lcl|NZ_CP026520.1_prot_WP_042229067.1_1574 [locus_tag=PC41400_RS08155] [protein=SDR family oxidoreductase] [protein_id=WP_042229067.1] [location=1790575..1791432] [gbkey=CDS]
MEHKIMMVTGANSGMGLATTIALARQGAKVVMVCRSEERGQSALREAIASSGSGEIELMQCDLGSLRSIRSFAADFKSRY
DHLDVLINNAGVVSLKRETTSDGFEVMMGVNHLGHFLLTNLLLGPLKRAEQGRIVVVSSGAHKIGKIRWEDPHLTKGYTA
WTGYAQSKLANILFTKELAVRLKGTAVTVNALHPGAVGTQIGVDRNTGFGKSVLAMLRPFFLTPAQGAETAVYLATSGNV
SCATGEYFYRKKIAPVSARAKDKELAAKFWDWSAGEVGLQPSETE
>lcl|NZ_CP026520.1_prot_WP_042229066.1_1575 [locus_tag=PC41400_RS08160] [protein=BglG family transcription antiterminator] [protein_id=WP_042229066.1] [location=1791778..1793874] [gbkey=CDS]
MTLLQQAADYVSLEELMQKLGISRRTVYYDMEKINDWLKTIHLPPVEYVRAAGYYLPDESRKGIPELAGQLPVQQYYLSP
RERLAWLVVHLLSPAGGSLFVHHLTDLLQVSRGTVLKDLNIVRPELEALKLKLTYDRKQGYRVRGEESDKRKALAYYFGQ
IFGSVGWEQALSRSEDPGSFDRVLNLFPPMKPDHLKRVYRLIADSERTLGVELTDEMVFNLTTRFLLYTRRIQNKQLISI
DEDERSVLAHMPEYEAAARIAEGLGYLFDTAIPDEEIAYFTMHLLGARVNRMEQMTGSEEVIDKLREAAKGMTDAFQRHA
CVFLPNREAMEENLLVHLKPAYYRILYGLGLDNPLLGNIRSKYGEVFELTRRSAAPFEQLVGRRLNDNEIGYLAMHFGGW
LRRENTEPVERRRAAIVCVNGISASRMLKSQLEHLFSAVDITAVLSLRDYEAFEEESVDFIFSTVPLPDSVVPVFVVNPI
LSDKDKEHLLNQINSVVGRKSRAQASSVQALLDIVRKYAKVTDEAGLTEELNRYMSVGKTWLSEARKPSLADLLTGPMVR
VEKEAADWRSAIREAALPLLEDGRIEEAYIGAMIAKVEALGPYIVVAPGIAIAHGKPQEGVKRLGISLLRLAKPVAFSEE
ARHQVQVILVLAAIDGESHLKALSELTLLLREQDSLTRLKDAGTKEQILQQIQEGPRS
>lcl|NZ_CP026520.1_prot_WP_042229065.1_1576 [locus_tag=PC41400_RS08165] [protein=PTS sugar transporter subunit IIA] [protein_id=WP_042229065.1] [location=1793901..1794344] [gbkey=CDS]
MKFLEPSLIALDVEAETADEAIRLAGGLLADAGAAEARYAEAMVDSYREKGPYFVLAPHIALPHARAEDGVKEASVSLLR
LKKPVVFGHAANDPVQLVFALGGSTSSEHIAMLRKLTMLLSNPDNVEAFKQAADVDTIQQLIRGNEQ
>lcl|NZ_CP026520.1_prot_WP_042229064.1_1577 [locus_tag=PC41400_RS08170] [protein=PTS sugar transporter subunit IIB] [protein_id=WP_042229064.1] [location=1794341..1794604] [gbkey=CDS]
MKILCVCGLGQGTSLILRMNVENVLSGMGVSADVEHTDVSTASGTAAEFIITSNELAQSLQGHSAKIVIVNNYFDNDEIK
QKLEEVL
>lcl|NZ_CP026520.1_prot_WP_042229063.1_1578 [locus_tag=PC41400_RS08175] [protein=PTS ascorbate transporter subunit IIC] [protein_id=WP_042229063.1] [location=1794625..1795920] [gbkey=CDS]
MDVLFWIATNIFGTPAILLGFIVLVGLLLQKKSTSQLVSGTFKAIIGFLVIGAGSDVIVKALNVFEPMWKEVFGLEAGSL
GSFMGQEAFNGKYGSAVTLAMLVGFLINVALARFTRFKFIYLTGHMMFWTSTIFAGVIVHASSGIAIWKLVLMIAVVLGL
YWTFQPALVQPFLRKVTGNDNIALGHTSASVALLAALFGKLLGNKNNDSEKIKVPKGFEFLRDSNVITALSMGILFLIGA
IILSTKGTPAAAELMAKSGDQNFIIYSIIQSFTFAGGIAIVLMGVRMFVGELVPAFNGIATKLVPGAKPALDAPVVYPYA
PNAVVLGFLGAFAGAIIWLLVLGNTVSYIFVPTMIVLFFHGATAGVFGNSTGGVRGAVIGGFITATIVAWGQFLMVRFFI
SSTVPDTAMWAADSDMFIIGPILNFLAKLFF
>lcl|NZ_CP026520.1_prot_WP_042229062.1_1579 [locus_tag=PC41400_RS08180] [protein=phosphotriesterase] [protein_id=WP_042229062.1] [location=1796020..1797012] [gbkey=CDS]
MSYIRTLHGDIAKEDLGFTYSHEHIVCRPAFWQERGEDDLLLDDKEKSKRDVLDFKKHGGHAIVDATAVDYGRDVQAVQE
IALETGIKIVGTAGFNKSFLWDAKIKEELKPLTGDYETYTQWIEAKSINELADFVIREVEEGLEGTPFKAGQVKFGTGYN
RITPLEEKTLRAVARAHHETKAPVHSHTEVGTMALEQIELLKSEGVNLSYLSLGHMDRNPDPYYHEQIAATGAYMSFDGI
GKIKYAPESTRIALILNLVSKGYEDQILISGDTARKTYYKHYDHGLGLEYIIAKWVPRFVDEANRNGFDGQKLIEKFFVS
NPANCFSFKK
>lcl|NZ_CP026520.1_prot_WP_042229059.1_1580 [locus_tag=PC41400_RS08185] [protein=creatininase family protein] [protein_id=WP_042229059.1] [location=1797026..1797784] [gbkey=CDS]
MKFEHENSYEVQDKIRQFRAAILPIGAVEAHGPHLPLGTDNLLAERLADKLAARTDSFVLPVLPYGQVWSLRNFPGSINV
SNEALIRLLADIGESLYQQGFHIFTMVNGHLGNAVALKEAARVLYERVPELKVFYFFYPGTKEITAQVRESSAAHGTYMH
ADEIETSYMLYLAGEYVDMSKAIDGAPRITAEADCTPTPWEEITSSAVLGDATLATREKGEKIIERSLDVMAEMLLRTKR
ALEDGDHQDSTD
>lcl|NZ_CP026520.1_prot_WP_042229907.1_1581 [locus_tag=PC41400_RS08190] [protein=SIS domain-containing protein] [protein_id=WP_042229907.1] [location=1797804..1798538] [gbkey=CDS]
MMNTYFDVIGELLNKVRTEQAGPIREAAEKVAACIQDGGIIHLFGCGHSHILTEEVFYRAGGLVPIRPIFVEKLMLHEGA
VRSSELERQNDLAVDFMKEQDIRKGEVVFVLSTSGRNPVPVDVALMAKEKGAVTVGITSLEYSMSQPSRHTSGKHLSEAV
DLVIDNFSVKGDAVLNHPGLQVPFTPSSTVIGAAILNAVFAESIEIMIENGFNPPVFLSGNIDGSDAHNQALIQTYKERI
ALLA
>lcl|NZ_CP026520.1_prot_WP_042229057.1_1582 [locus_tag=PC41400_RS08195] [protein=HPr family phosphocarrier protein] [protein_id=WP_042229057.1] [location=1798798..1799067] [gbkey=CDS]
MFAQTVTIQNAQGFHVRPAQLFAETAGQFGAEVHLKTGDGRTADCKSMLELMTLGIEKGAVVTIEASGDGEKEAVEALAQ
LVESKFGEA
>lcl|NZ_CP026520.1_prot_WP_042229055.1_1583 [gene=ptsP] [locus_tag=PC41400_RS08200] [protein=phosphoenolpyruvate--protein phosphotransferase] [protein_id=WP_042229055.1] [location=1799067..1800824] [gbkey=CDS]
MKGLHIQGIGVSEGIRIAQAFVYRAAHGNAGGASAEWKPIAADPAVVEAELAKLKEAKARCAGELERLTEKAKETVGEEQ
AGILAGQRKLLDDPAFYPKMEARVREKLEPAPQAVSEIVSQVASWFEGMTNDYMKERAADIRDIGSRLLACLSPGERAGL
EGITGEVILVADDLTPSDTVQLDPAKVVGFITRTGGATSHTAILARSLGIAAIVGAGDAVDGIDDGDTLVVDGAAGLCIA
RPGEAEAEQYRLRMGQDLERRKSLEAYAARPAVTADGTRVELAANIGTAAEAGAAAAQGADGVGLYRTEFLFMNSPSMPG
EEEQYRAYLEAAKLMEGRPVIIRTMDIGGDKELPYLQLPKEVNPFLGYRAIRIGLSRKELLLTQLRAVLRASAYSKVKVM
FPMISGLQEWRQAKEILAEAQAELRQEGVPFDENMEVGIMVEIPSTAVMAGQFAREVDFFSIGTNDLVQYTLAVDRMNEH
VAYLYDYFHPAVLQLIRTVIEASHSAGKWTGMCGGMAGDPLAAPLLLGLGLDEWSMEAGRLSPLKEVLAGLDRQECRELA
ERVLLLDTPQAVRQALEEFRESKSV
>lcl|NZ_CP026520.1_prot_WP_042229053.1_1584 [locus_tag=PC41400_RS08205] [protein=pyridoxamine 5'-phosphate oxidase family protein] [protein_id=WP_042229053.1] [location=complement(1801306..1801926)] [gbkey=CDS]
MRRKEFGVEDWQEIEQFMSEMTFGILGTIGEDGYPNLKALNFVYADGAIYAHGSKVGEKISDLGRNPLVTFAIAKEYALI
PSYFTEPVYACPATAFFKSVVIKGKAVLVEDLEEKAKALGVFMEKLQPEGGYEPFSLENRGYVSQIRGVAVIRIDVEDLS
AKFKFGQNMKGAKREAVVNGLENRGKELDEETVRLMKLYCPAHREG
>lcl|NZ_CP026520.1_prot_WP_042229051.1_1585 [locus_tag=PC41400_RS08210] [protein=Gfo/Idh/MocA family oxidoreductase] [protein_id=WP_042229051.1] [location=1802219..1803214] [gbkey=CDS]
MLKVALLSFWHVHALDYEKDANQHPDTQVAAIWDEDPARGRKEAENRGIPFYENLAELLADDEIDGIIVTTPTHLHRDVM
VAAAKAGKHIFTEKVVAATKKESDEIMDAVREAGVVMTVSLPRLNAPYTLAIRDILEQGVLGKLTLVRTRLSHNGGLSRE
EGGGWLPERFFDESAAQGGAMIDLGCHPMYLTRLFLGMPESLTANYGSVTGRSAEDNAVSVLRYPNGALGIVEAGFVNSH
SPFVIEIHGTQGTLLYSQHDDKLLLRTRAQEGEEWQTVTDLPQAKPSAFEQWVAHIESGTRAEENLLLAADLTRLMEASN
ASAASGCAVKL
>lcl|NZ_CP026520.1_prot_WP_042229049.1_1586 [locus_tag=PC41400_RS08220] [protein=ester cyclase] [protein_id=WP_042229049.1] [location=complement(1803802..1804272)] [gbkey=CDS]
MKALRPEEVVKRFFDEVRSGKKLAAAPELMAGKVFAHQVQSEQEVTVERTPEVYAEHVQEMLDAYGPFTLEVQELLGSGD
RVYVRWKQTGTHIGEVDGYPPTGLPVIELASAVYRVENGKIAEYWIQIDRAGIERQLKRNAAEAGAATATAGGAAD
>lcl|NZ_CP026520.1_prot_WP_042229048.1_1587 [locus_tag=PC41400_RS08225] [protein=helix-turn-helix domain-containing protein] [protein_id=WP_042229048.1] [location=complement(1804360..1804725)] [gbkey=CDS]
MTARSGKTPAEPAANSYIPKEPCVIECSIEKTLDVIGGKWAFLVIRELFDGTRRFGELQRSIASISPTALTNILRHLEKQ
GVLERKAFATVPVTVEYTLTPKGRDLHKILKEMKLWAAKWT
>lcl|NZ_CP026520.1_prot_WP_042229046.1_1588 [locus_tag=PC41400_RS08230] [protein=aldo/keto reductase family protein] [protein_id=WP_042229046.1] [location=1804922..1805881] [gbkey=CDS]
MHYRRLGRSGLKVSEISLGSWLTYGGYVERDNAVQSIRKAYDLGVNFFDTANVYKQGEAEKVVGDTLKEYPRESYVLATK
AFWPMGEGPNDRGLSRKHIMEQAHASLKRLGTDYVDIYYCHRYDPETPLEETLRALDDLITQGKVLYIGVSEWTAAQMAD
ALGLAERYLLDKIVVNQPVYNMFNRYIEKDIIPLGEEKGIGQVVFSPLAQGLLTGKYKSADDIPADSRAAKVETMRGAVT
AENIAKVKQLEGIAAELDLTVGQLALAWILRQPNVASALVGASRPEQVEQNVKAEGVKLAADVLERIEAVLSGNEAAGK
>lcl|NZ_CP026520.1_prot_WP_241688819.1_1589 [locus_tag=PC41400_RS30210] [protein=hypothetical protein] [protein_id=WP_241688819.1] [location=complement(1806031..1806255)] [gbkey=CDS]
MKEAFAKDKAELQGLLPQLQATLNTHTTAAFGLVFKVSCLALLLGAVCAFVNGRKAEEPAREAERVENALRSPQ
>lcl|NZ_CP026520.1_prot_WP_241688820.1_1590 [locus_tag=PC41400_RS08235] [protein=DHA2 family efflux MFS transporter permease subunit] [protein_id=WP_241688820.1] [location=complement(1806230..1807729)] [gbkey=CDS]
MQKSRKIPAILTLLLGFFMAILDTSIVNVALPRMSAFYGADTAGISWVVNGYSLSFAVFLLAASRLADQFGRKKAFMTGL
FLFTLSSLLCGLAGSITSLIVFRVLQGIGAALITPLVAPLLLALFPPEKKAVMMGITGALAGLAAASGPALGGVLSYQLD
WQWIFYINIPIGALALVLAASSLAESYDPTATRKLDWGGMLTVSAASLTLTLAFMQANEKGWDSPYILSLFAAALVSLLL
FLWIESRVKEPMLPLSLFRIKRFPSGNLGLFLLGIGMTGPTFILAFFLTQVIGMTELRAGLIISVLAIASMICSVLAAKG
AAKLGPRIFSVTDDPACPRNLPAGGTHDRVLDDGLHLAPGHCRNGNRHGDREFDGNYRPASSAGKNDHRHGHRHHGQNDR
ERARHRRSRHSAVPIHADRDGLRQNRAGGTDRRQLSVRGEQGGTCRSSAAAGRKPQGCFGKRLRVDAGSARPMGPQGAGT
ASSRVPRCGRAGEGSLRQR
>lcl|NZ_CP026520.1_prot_WP_042229043.1_1591 [locus_tag=PC41400_RS08240] [protein=TetR/AcrR family transcriptional regulator] [protein_id=WP_042229043.1] [location=1807890..1808573] [gbkey=CDS]
MEKLARVTKSPEERKQEFLSAAESLFRVKGYERTAVSDIVKQVGVAQGTFYYYFKSKDELADAMIRRMLDEHMGLIESIA
EDPELSGQEKIVKVMKEDFVKRGDNLENLDYMHHENNALLHQKMLVESVKAYTPFITGIIEQGIGEGSFHTSHPREVVEF
FMVGLFFLYDPGIFAWTVEEVVTKLQAIGEILEKLLGAEKDSLDIWKDLPDMPIIIQKALKAEKASR
>lcl|NZ_CP026520.1_prot_WP_042229041.1_1592 [gene=mtnA] [locus_tag=PC41400_RS08245] [protein=S-methyl-5-thioribose-1-phosphate isomerase] [protein_id=WP_042229041.1] [location=complement(1809007..1810116)] [gbkey=CDS]
MTANENGVQSEGQRGWVQSVRWNGERLDLLDQRLLPEEIVYLPLETPHEVWEAIRHLKVRGAPAIGIAAAYGVFLGIRDA
QEDESISREANRRRLAGEVAKQADYLATSRPTAVNLFWALDRLKRLTQKLVEMDISHGEMKQAVLTEAISIQAEDEETNR
RIGEFALELFEDGMGVLTHCNAGGLATAKYGTALAPFYLAQEKGVSLKVYADETRPVLQGARLTAFELQQAGVDVTLICD
NMAGAVMKKNWVQAVIVGTDRVAANGDVANKIGTYSVAVLAKAHGIPFYVACPMSTIDLSTPTGEDIPIEERGEEEITQG
FGKRTAPQGVKVYNPAFDVTPNEYVTAIITEKGIVRAPFAENLAKLFEE
>lcl|NZ_CP026520.1_prot_WP_042229040.1_1593 [gene=mtnK] [locus_tag=PC41400_RS08250] [protein=S-methyl-5-thioribose kinase] [protein_id=WP_042229040.1] [location=complement(1810116..1811342)] [gbkey=CDS]
MTTTYHALTEAEAIAYAQGLPNIFSPDAVLTSREIGDGNLNLVFHISDEKSGRSLILKQALPYAKVVGESWPLTLDRARI
ESEALQLQGSLCPDLVPHVFAYEKELALTVMEDLSDHVIMRKGLIEKNKYPLFAKHIGRFMARTLFFSSDLGMNQQEKKL
RVGRFINPELCKITEDLIFDDPYRDSENNNIEDHLKDAVQAIWQDRELHLEVAILRNTFLTRAEALLHGDLHTGSIFIKE
DSTKVIDPEFAYYGPMGFDIGAVLANLLLNYAGQEGWSAGETERTYYREYLLETVKEVWTVFEREFRALWDNETVDPMSE
TPAYQDWYMNRVLQETFGFAGCKMVRRIYGLSHVADIDGIEDAAARERAQRLALAIGTTLIKRGRQASSIDELIAVAREA
SVAKEGVR
>lcl|NZ_CP026520.1_prot_WP_042229039.1_1594 [locus_tag=PC41400_RS08255] [protein=DNA starvation/stationary phase protection protein] [protein_id=WP_042229039.1] [location=1811544..1811993] [gbkey=CDS]
MSIAIRSAHKVLNKQVANWTVLYVKLHNYHWYVKGNQFFTLHVKFEELYNEAAGYVDELAERLLSIGGKPVATLKECLEL
SGIGEAAGSENSDAMVRALLADFETIISECKTGMKEAEEAGDEGTADMLLGITQSLEKHRWMLGSFLNG
>lcl|NZ_CP026520.1_prot_WP_042229037.1_1595 [locus_tag=PC41400_RS08260] [protein=DUF1802 family protein] [protein_id=WP_042229037.1] [location=1812239..1812832] [gbkey=CDS]
MTHSHMALKEWASAVQALEQGEQVLIMRKGGIREETRDFRVVSDAFYLYPTYEHQKPHLIKDSAQPRLQETLDGWNPDDR
QVTISSWAELVRDIEISDGETLEKLQPFHIWTETFAEERLKWKRKQPLHVMLLRVYKLDKPVTLDVDPSYEGCKSWIELH
GELESAGRTPVLSDEAFRAAVDRVEKALADGGPVNGA
>lcl|NZ_CP026520.1_prot_WP_042229035.1_1596 [gene=sufC] [locus_tag=PC41400_RS08265] [protein=Fe-S cluster assembly ATPase SufC] [protein_id=WP_042229035.1] [location=1813173..1814009] [gbkey=CDS]
MANAPVFTIEGLKAQIEGKEILKGLSLEVKGGEIHAIMGPNGTGKSTLASTIMGHPKYEVTDGSISLDGEDVLEMAVDER
ARAGLFLAMQYPSEIAGVTNADFLRSAINARREEGSEISLIKFIRQMESKMKELEMNPEFMHRYLNEGFSGGEKKRNEIL
QMLMLDPRIVILDEIDSGLDIDALRIVATGVNAMRSEDRGFLIITHYQRLLDYVTPDFVHVMMQGRIVKSGGPELAQRLE
KDGYDWIKEELGIVDETVGNDDEQEFQVPMNTTAPKYN
>lcl|NZ_CP026520.1_prot_WP_042229034.1_1597 [gene=sufD] [locus_tag=PC41400_RS08270] [protein=Fe-S cluster assembly protein SufD] [protein_id=WP_042229034.1] [location=1814042..1815337] [gbkey=CDS]
MTTQTILPVDSQAVAQASKARQEPEWLTDLRVQAANLAGSLELPKLEKTRLDRWDLSSFGTYREPQALASVSELPEQVKT
LVGDTNIIVQRNSGAVYQTLSEELAGRGVIFTDLETAVREHADLVRPYLGKAVEAGENLLTAVHAATWSGGVFLYVPKNV
NVEIPLQALFLSDDSAASFSPHILIVADTNSSVTYVENFTSIGTGASLVQNGVTEVFVKPGANVHFASVHNLTEGVTDLT
YRRAVVENDASVNWIVGEMNQGNSMSDITSILKGNGSYSDAKVICVGTNDQKLNLTTRAIHFGLSSNSDMITRAVMRDQA
TAIINGITKIEKGATGANGQQTERVLMLSPKARGDANPILLIDEDDVKAGHAASVGQVNQDQVYYMMSRGISREEATRLL
IYGFLAPVVSEIPIAKLEEQLQTMVERKLGQ
>lcl|NZ_CP026520.1_prot_WP_042229032.1_1598 [locus_tag=PC41400_RS08275] [protein=cysteine desulfurase] [protein_id=WP_042229032.1] [location=1815334..1816560] [gbkey=CDS]
MKPSEIRELFPILNQEVNGHPLVYLDSSATSQKPVQVIEALKKYYEWDNSNVHRGVHTLGSRATDAYEGARAKVVRFINA
RSEQEIIFTRGTTTAINLVASSYARSICGEGDEIVITPMEHHSNLIPWQQVAKATGATLKYIPLQPDGSIALEDVETTIT
DRTKFVSVVHLSNVLGVLNPVKAIAEIAHRHGAKMMVDGAQSAPHLKVDVQDLDCDFFAFSGHKMCGPTGIGVLYGKKEL
LEAMEPIEFGGEMIDHVDLYDSTWKELPWKFEGGTPIIAGAVGLGAAIDFLEEIGLDAIADHDASLTAYALERMLEVEGL
NVYGPRNNRVGLVTFNLGDVHPHDVATVLDAEGIAVRAGHHCCQPLMRWLEQSATARASFYLYNTEDDIDRLVQGLLKTK
EYFGHAIG
>lcl|NZ_CP026520.1_prot_WP_042229031.1_1599 [locus_tag=PC41400_RS08280] [protein=SUF system NifU family Fe-S cluster assembly protein] [protein_id=WP_042229031.1] [location=1816547..1816975] [gbkey=CDS]
MQLDDLYRRVIMDHYKTPRNRGSFEDNESVSINLNNPTCGDRITLQMQVEDGKVKAAKYVGEGCSISMSSASMMTDAVKG
KTLEEALEMAENFSKLMKGEDVGFEYEDIEALSGVNKFPARIKCATLPWNALRKGIEQTKQS
>lcl|NZ_CP026520.1_prot_WP_042229030.1_1600 [gene=sufB] [locus_tag=PC41400_RS08285] [protein=Fe-S cluster assembly protein SufB] [protein_id=WP_042229030.1] [location=1817004..1818401] [gbkey=CDS]
MAKQMPEMEEYKYGFRDEHKSIFQSGKGLNAEIVTEISRMKGEPEWMLEFRLKSLEQFFSMPMPRWGGNLDDLDFNDIQY
YVKPSEKQGKTWEEVPTEIKETFDKLGIPEAEQKFLAGVSAQYESEVVYHSMQKELEDQGVIFMDTDSALREYPELFREY
FGTVIPPNDNKFAALNSAVWSGGSFIYVPKGVKCEVPLQAYFRINSENMGQFERTLIITDEDSFVHYVEGCTAPVYSTNS
LHSAVVEIVCRKNSRARYTTIQNWAPNIYNLVTKRAVAEEGANMEWVDGNIGSKLTMKYPAVVLKGRGAKGSVLSIAVAG
KGQHQDAGAKMIHLAPDTTSTIISKSISKHGGKVTYRGLASFGRNSAGAKSNIKCDTLIMDKQSTSDTIPYNEIMNDNIT
LEHEATVSKVSEDQLFYLMSRGLSEGEATQMIVMGFIEPFTKELPMEYAVEMNRLIKFEMEGSIG
>lcl|NZ_CP026520.1_prot_WP_042229029.1_1601 [locus_tag=PC41400_RS08290] [protein=hypothetical protein] [protein_id=WP_042229029.1] [location=1818641..1818955] [gbkey=CDS]
MKQYGMLFTLVVLLLTGCSTDKQTYTLANIQLITEGSRIIQSDGNKLVMLEYHFTIKGSSAIDTQSAEISYGYTGHSKDV
VPINTGTLSLKSLCKRFSGGDGCK
>lcl|NZ_CP026520.1_prot_WP_042229028.1_1602 [locus_tag=PC41400_RS30215] [protein=hypothetical protein] [protein_id=WP_042229028.1] [location=complement(1819126..1819698)] [gbkey=CDS]
MADERNKSTHPSNEQPLVTIKRKSDPAEESASFETDRAESRRGHEPPSPDSGGERSLAPSAPEPVTFRATEEFPVHPGHL
DSASDSERRPESFFPPDGAPPTIRKKRTSVVLSIVCGILAILLNLPVVTLPLGLILGFVAFGAGLAARRAHRRSAGLVLS
LFSFLIAAVWLASALVPLAVDPTISVGRLP
>lcl|NZ_CP026520.1_prot_WP_042229027.1_1603 [locus_tag=PC41400_RS08300] [protein=HD-GYP domain-containing protein] [protein_id=WP_042229027.1] [location=complement(1819864..1820922)] [gbkey=CDS]
MRIHISDLQVGDRLTRDTFNSIGLNILSADTKVDEFDILKLNKHQIEYVDIEERLRPGHEAQTETAVTAELSKQEESFHT
AIGGIKDLFESAVRTGKLSDTVVEQYYTPLISNVREEKDVVNLLLCLDSRDDYTYQHSVQVGLISYFISTWLGKSEEEAK
QIGKAGYLHDIGKCRISFDILQKPGKLTNEEYAEMQKHTVYGYEIIRNSIADELSALVALQHHERLNGKGYPLGKTGEAI
HPAAKIVAVADVYSAMINNRSYQSKRDLLYVLKELHKMSFGELDPQAVHVFIANMIPNFIGKKATLSNGQSGTIVMTFPN
DYFRPLIQIDDAEDTFIDLTQRTDLEIQSITV
>lcl|NZ_CP026520.1_prot_WP_174446804.1_1604 [locus_tag=PC41400_RS08305] [protein=ThiF family adenylyltransferase] [protein_id=WP_174446804.1] [location=complement(1821187..1822215)] [gbkey=CDS]
MPARYSRHVLFKPVGENGQRALGRARVAIVGLGALGTVLANHMVRAGVGFVRLIDRDFVEESNLQRQMLYDEEDAHQAQP
KASAAAAKLRLVNSLVGIEAHVADLNPLNAEELLTGVDLILDGTDNFSVRFLINDVSVKHGIPWLYGGAVASRGVSLAIL
PGETPCLRCLFGSAPAHGTADTCDTAGVIGPIIHMVASYQATEAMKILIGDRQHLNRRMQQWDLWYNHYTSVDVSGARKA
DCPACAHRSFEYLDAEIDGETIQTLCGRDSVQIHPVRPADFSLEHWEQRLAPLGRVSRNKFLLRFRPSDDITLAVFPDGR
VLVQGVTDPVAAKTLYSRYIGM
>lcl|NZ_CP026520.1_prot_WP_042229023.1_1605 [locus_tag=PC41400_RS08310] [protein=molybdenum cofactor biosynthesis protein MoaE] [protein_id=WP_042229023.1] [location=complement(1822622..1823386)] [gbkey=CDS]
MTFHIRLFAGLADELGRPSIDIALSPDASGRSATVQDLLDEIAARFPAARSLLAHCFLAKNHAYAAPHELVSESDELALI
PPVSGGSGEAPDAGDEAAEPARYVLTREPIDPAEVIQKVTHPDCGATLAFIGTTREHTHGVRTVLLEYEAYEPMAVSVME
TIGAELEARWPGCRCAMTHRLGPIGIGESSLVIAVSTPHRADCYEANRYAIERVKQILPVWKKEIGEDGSEWKGYVSPEG
GGGWNPLDPPPGRE
>lcl|NZ_CP026520.1_prot_WP_042229022.1_1606 [locus_tag=PC41400_RS08315] [protein=bifunctional UDP-sugar hydrolase/5'-nucleotidase] [protein_id=WP_042229022.1] [location=1823543..1824994] [gbkey=CDS]
MPEKQQKHQTLVILHTNDLHSHFEQMPKIATCFKELRRRHEGNEILTLDIGDHLDRMRLETEGTGGAANIEIMNATGYEA
MVPGNNEGLTYAPDALRSLFETKAEFPVIGSNILDADTGKVPAWMIPYHIVRKGGITVGLIGLTAAFTDYYELLGWDVQE
PIKAAAYWVEKLRPQVDIVIVLSHLGLRYDERMAAEIEGIDVILGAHTHHLLEKPLHIRSTLVCAAGKYGQYAGEVEIAY
NVRERRIERLAGRVHDVSGCEDDAEIAGILARNAELGRRALDIPVTDLAEPLELDWYGESPLGNLLASGLRRWMNAEIGL
VNAGQLLRGLARGPVTLGRLLAVCPGPINPCRVKLTGSRLLQALEESLLPEFQSKPIRGFGFRGEVLGILCLDGLTVRYD
PEGKPLRKIREVLVNGEALDPERSYTVGMIDMFTFGVGYLSLHEATDVEYLLPELLRDILGEELQHPERFQEAFESRWQP
EGI
>lcl|NZ_CP026520.1_prot_WP_042229020.1_1607 [locus_tag=PC41400_RS08320] [protein=undecaprenyl-diphosphate phosphatase] [protein_id=WP_042229020.1] [location=1825109..1825939] [gbkey=CDS]
MFSLWIAAIIGIVEGLTEFLPVSSTGHMILTQSLLGISDDEPLMKTFTIVIQLGAILAVVIVYWKRILRILGINREPVAV
KAGASRLNVLHILMGIVPAMGIAFLFDDLIETYLFSPKTVVVSLILGGILMIIAEKKMDRTKGPVVEDMDGISYRQAFII
GLYQCLSVLIPGFSRSGATMAGGMLHGVTRKAGADFTFIMAVPIMFAASGYSLLKSYKNFTSDDILFFAVGFIVSFIVAL
LAIVTFIRFVQRMKLTYFSYYRFALAAIFIVYLWLT
>lcl|NZ_CP026520.1_prot_WP_042229019.1_1608 [locus_tag=PC41400_RS08325] [protein=HD-GYP domain-containing protein] [protein_id=WP_042229019.1] [location=1826130..1827218] [gbkey=CDS]
MRLLPISQCQPGMRLGRHIYNEQGAVLLAENVELTQSLLERLKTYGIDYLYIQDALTEDIVPDDLVSEETKLRLFTEIRS
TFRKLMNEPGRRTLANYQHFGKSFRQQMDHVIDDLSRHKDAMIMLGDIQIKDHYLYQHSLNVCLYTTMLGMSHGYERDEL
MTLGLGALLHDIGKTQISEEILSKPSKLTEPEWSEMEKHTVYGYQMLKNEANIPLLSAHCAFQHHERINGSGYPRGIKDE
EIHPYAKWIAMVDSYDAMTTQRVYRKAMLPHEALEVIMGGAGTLYDTDMLNLFRDKVAIYPLGVTVALNTGEIAVVVDLN
ASAPQRPVVRVIKNEARELIASPYEIDLSKKLTIMITEVHPL
>lcl|NZ_CP026520.1_prot_WP_241688821.1_1609 [locus_tag=PC41400_RS08330] [protein=MMPL family transporter] [protein_id=WP_241688821.1] [location=1827441..1829696] [gbkey=CDS]
MNGMNGMKPSAKSELFMQWGYAMYRYRKRVLAVWILFFAGAAILAGQVPALLKDDGFTPKGSESDRGFHLLQEKLGVPPS
SLQLVYDGNGKDLTTEKAKASIAKSLEPLKSLPYVDTWAFVESPRKESARAAEIQAVNVALKLEVEKVLEKYGEIQARLT
PPPGMNVYTGGAPALLADMQKASKNDIAKAEMIGLPIALIVLLIIFGTLTAALLPLIVGVASLTATLGITYFIARQVALS
NFLPNMVTMLGLAVGIDYALFMVSRFREELRAGREIPDAVAVTCRMAGRSIFFSGIAVLIGLCGMLFVDLSFFRSLCLGG
ILVVTISVLAANTLLPALLGMLGRHVNSLRVLPKAWSERGTTRRFWERTAYFVMRHPVVLALVLTAALTAMMLPIRGMQL
STPSGDVLPPQYGARQAGDLAGQAFDERELNPLQILATAPEGAWDAASVGRLRAYIDSVKATGGVKGVRSVLDGFPGLSD
EQAASVLSEPSLSSPGTGKTAAARDSVLIRVVPESDPDDRATDALVRSLRGLDRQRLDVHMTGGPAYRLDMLDRIQDKLP
VVIMFVLGVTFIVLLLAFRSLLLPLKAVLMNVLSLGASIGAVVLVFQHGFLADLMRVTSTGYVSATVPIVIFCVVFGISM
DYEVFLISRIAEEYDRTGDNERSTAEGMMKTGSLITSAAFILIVVVGTFIFTDIEIMKALGLGLALAVLIDATVVRVLLV
PALMKLLGRLNWWAPGWLRPHGPREGSDPLL
>lcl|NZ_CP026520.1_prot_WP_241688822.1_1610 [gene=yfkAB] [locus_tag=PC41400_RS08335] [protein=radical SAM/CxCxxxxC motif protein YfkAB] [protein_id=WP_241688822.1] [location=1829820..1830950] [gbkey=CDS]
MNERTAVPALKKLSPSYDPWDPIRSLQQHGRHTLTSVEFTVSNLCNMRCEHCAVGDTLVMKEGPKLALDHALRRLEEIQH
LETISITGGEPMFSEKTVKEYIVPLLRYARERGVRSQINSNLTMELERYEWIAPYLDVMHISFNYTCAEDFHEVGFVRSG
RPVGMKAASKMYDRMLDNSVALSKGGMFISAESMLNTRTSEKMVPIHKIIEEIGCQRHEVHPMYPSSFASNLPILTLDQL
RDTIHRLLDGRNPEMWMLLGTLPFYACNSSEEDRKLLSRLRSEPNLTVRNDPDGRNRLNVNMFTGDVYVTDFADVPALGN
LHEDKLETVFDRWLNHPLSRTVDCHCPAVSCCGPNLLVANSYYSGEDFTRRQAILT
>lcl|NZ_CP026520.1_prot_WP_174446803.1_1611 [locus_tag=PC41400_RS08340] [protein=hemolysin family protein] [protein_id=WP_174446803.1] [location=1831014..1832327] [gbkey=CDS]
MSLVLLLVLLNGFFVAAEFALVKVRQSKLQQLVNEGNKKAKFAFTVTKKLDAYLSATQLGITLASLGLGWVGEKAIAELI
IEPLFHALNWHSTFLSGTISLVLSFSIVTFLHIVLGELAPKSLAIQKPDAVSLWTSAPLMYFYKIFLPAIWVLNGAARLL
LKMFGVQPANEHDAAHTEEEIRILMNQSAKSGHIDRDEMLLFDNIFEFSDRIAREVMLPRTDMDCLFIDQPYDEILQHIY
ETKHTRYPVALGDKDQIIGFVHITDLLTADATVQVELQQFVRPILSVPESMEISQVLKLMQKKHSQLAIVVDEYGGTAGM
LALEDIMEEIVGELHDEFDEDERPSIEVKGNITSVEGRTLIEDLNVLLHLEIEDEEVDTIGGWVFKTLEGSPIKGKKVVF
GGHTFEVAEVDRLRITRVNITKNESETPKDPAAGLEE
>lcl|NZ_CP026520.1_prot_WP_042229016.1_1612 [gene=ytvI] [locus_tag=PC41400_RS08345] [protein=sporulation integral membrane protein YtvI] [protein_id=WP_042229016.1] [location=1832345..1833391] [gbkey=CDS]
MSWKTLLILAFGLLLLYGLFTVGMPFLLAIVTAMLLEPVVTVGMRLLKLRRFAAVSLVSTLFTLLTLGLFSMIGVKVVTE
FMNFWGKIPVYLEETGTYVEDAAARARLMYESLFPGDNSQMQVWLQNGLASAAQALERFILLLSDSFLGAAAAIPNLLIL
FIVYLFSLYLFSYSLPVLKQSFLNMFDDRSREKVEKLILSLRQALFGFMYAQVMMSMLTYVLTLIGLLFIRTDYPLAIAL
LVMFMEFVPVIGTGLVFLPWAIYRLLLGDPGMAMQIMLLFIVITLARRVIEPKILSNSVGISALAAMVSLYIGFELMGVA
GMFLGPLLVILFQTMRQVGLLQFKINLD
>lcl|NZ_CP026520.1_prot_WP_042229015.1_1613 [locus_tag=PC41400_RS08350] [protein=beta-N-acetylglucosaminidase domain-containing protein] [protein_id=WP_042229015.1] [location=complement(1833605..1836190)] [gbkey=CDS]
MTHHLKAPLLTPPPQQVEYAGYTVPLYATTGILAGTGTSPEALLELRGLLIGAGVKRVVPCKDWHSAAKMPLSVYIGKLP
ADPAVQQEFRSRRSGDPLGIPEEGYTLAARRSKHGGSMIVLAGADHTGTLYAVRTLKQLIRKGPMMANVPAVTIRDFPAF
PVRGVVEGFYGGPWSPDDRVNLIAFLGDHKMNTYLYAPKDDPYLRGRWREPYPEGELRHIRRYVDAAEKAGVRFVYALSP
GLDICFSSNEEFQKLAAKASQMWTLGVRHFSLLMDDIFQNPNCADDVQKFGGDESPAAAAQAYLLNRFNREFVQAHPGAS
RLITAPTEYYQDGSSPYRKRFAELVDPGILVCWTGIGIAPAVITSGEAERIHKVFNHDMLLWDNYPVTDYVREKLFLGPL
EGRDPELHKHGQFGYLSNPMEHLQASLIALFTAAEYAWKPEAYRPWESWERSLKEFGGSLHKELRILAENHLSSAVHTGE
SPNLAALLAAFWEDFEKGAPEKAAARLTAYFGSMRRAASRLTQMHNPDFVRETADWLKKIGFYGKAGESAVSLLLAMMRD
SGEEADARYSAYATVLAQDTVDIEVSEPRTGSRRIDGENRERGESELIRYTPVYGARTGTNEWGYEVTVVNGRIIREGGN
NSVIPADGYVLSLHSGQDREWLKSMTIVGAKVDIRDGRVTISVEKGTYPVPNKKVSADGVMEPFLQRVSSAYEMYVRWRE
NRGFTDPISTLPAWENNVLKNMTDGRADTYFWSGRAPQSGDYVGVNLGQTAQVRRVRISLGHPDPAAPQAGDLIRSASLE
ASLNGLSWTKLAEFTEQREIDFTPAAPFRAHFVRLKSSAVQTEWVMVREFDVTAERDQAAN
>lcl|NZ_CP026520.1_prot_WP_174446802.1_1614 [locus_tag=PC41400_RS08355] [protein=spore coat associated protein CotJA] [protein_id=WP_174446802.1] [location=1836415..1836600] [gbkey=CDS]
MYHPFVGPFDPCPPKLVKTYVTPPNLFIQFQPMCLPQFSPYEALRLGTLWPELYSSYEPKC
>lcl|NZ_CP026520.1_prot_WP_042229011.1_1615 [locus_tag=PC41400_RS08360] [protein=spore coat protein CotJB] [protein_id=WP_042229011.1] [location=1836639..1836905] [gbkey=CDS]
MHKQLDENYYRLLHDLQAADFVLVELTLYLDTHPEDAHALQQFNEFSQKRRQIALEYECLYGPLLQYGQNALPQQTWEWA
ETPWPWQV
>lcl|NZ_CP026520.1_prot_WP_042229009.1_1616 [locus_tag=PC41400_RS08365] [protein=manganese catalase family protein] [protein_id=WP_042229009.1] [location=1836983..1837552] [gbkey=CDS]
MWIYEKKLQYPVRVSKCDPHMAKLLIEQYGGADGELAAALRYLNQRYTIPDKVIGVLNDIGTEEFAHLEMIATMIYKLTK
DATPDQLKAAGLGDYYVNHDSALYYQNAAGVPWTAAYIQAKGDPIADLYEDIAAEEKARATYQWLIDMTDDVDLQDGLKF
LREREIVHSMRFREAVEILKDERAAQKIF
>lcl|NZ_CP026520.1_prot_WP_042229007.1_1617 [locus_tag=PC41400_RS08370] [protein=hypothetical protein] [protein_id=WP_042229007.1] [location=1837928..1838149] [gbkey=CDS]
MGITLSLLIGMLVLLAASLGLRKPKEFDRSDNLPFRPFDDIYEGKKRPTVESVQSQAVVKIEEKPRERKRIAK
>lcl|NZ_CP026520.1_prot_WP_042229005.1_1618 [locus_tag=PC41400_RS08375] [protein=3D domain-containing protein] [protein_id=WP_042229005.1] [location=1838616..1839581] [gbkey=CDS]
MALRHHWKKTCATLLLGAVVAASSAFQSQAAETETAKNDGLLYWKLAKSYQELAAGVTDTAGQAAGSELTQSFVTPENQI
LVLSEPNSMWNFASQMNMDVLDLLSLNPDADPLSLPAGSELKIVGEAALEQLKALKATVKVPVQAPAKVQLASATPSQTI
NIKPSEALQKTGKPQAPEPKKSAEKKAVKKPAVQAAAASAKSGQTLTTAAGKQIQYKKVLNIKASAYSADPSENGGYAGI
DYYGNKLTVGTIAVDPAVIPLGSTVYVTGYAFPGLPAKGMIAKATDIGGAIKGNRIDIFVPGSKDKVNNFGYQSVKVYIL
K
>lcl|NZ_CP026520.1_prot_WP_042229004.1_1619 [locus_tag=PC41400_RS08380] [protein=polysaccharide deacetylase family protein] [protein_id=WP_042229004.1] [location=complement(1839630..1840673)] [gbkey=CDS]
MKRAALAVSSAILLAFVSACSPAKLPDEGSASSRLDENRFIPHTLSEDPGPSSAPGPAKFQPVKVPAIKEKTAAAPSVSS
AAPLPTPSVPAKKLNNASIPKPEPKKKDVPVQKNHQEANAVQKTEQRKLSLAEIHHKYPTVFRMRGTSQGKKVALTFDDG
PDLNYTPQVLDILKKHQVKATFFVIGSRAQSHPEIIRRMVREGHVVGNHSYSHPFMPKLTASEFSSQIMRTQEVIKPLTG
YTPRLVRPPYGAINEEQVRWMSDHHLLAVNWDVDSLDWKGLSSKQVYHNIVDNARAGSIILQHSAGGNDKHDLSGTVTAL
PDLIRSLKSSGYKLVTVPELMGIPQDL
>lcl|NZ_CP026520.1_prot_WP_042229003.1_1620 [locus_tag=PC41400_RS08385] [protein=NlpC/P60 family protein] [protein_id=WP_042229003.1] [location=1840962..1841816] [gbkey=CDS]
MKIWVQSVVVSCAVLTAGLFAASDAHANAARSAKVQVNDQLVNTEGTAPFIDSSNNLMIPLRPVAEKLGTQIGWERAGTQ
MSISVTGNGHKITFTTGENQALVDGKKIDLPSKAQFLNNTVYVPMRFLADSLSVRVQWDNVNTIAIFDADGQYHAPAWYA
TVPQKSQKENTIVATANQYKGVRYVYGGSSPSGFDCSGFVQYVYDKNGVDLPRSSASMYASAGVKVSNPQPGDLVFFASN
SKMQHVGIYVGADKFISATDDGVKVDSLGSAYWGSRYVGAKRVM
>lcl|NZ_CP026520.1_prot_WP_042229001.1_1621 [locus_tag=PC41400_RS08390] [protein=phage holin family protein] [protein_id=WP_042229001.1] [location=1841948..1842343] [gbkey=CDS]
MNQTIFGSLAAFGASLVTYAFGGWSELLGFFLVAIAIDYVTGVAASLKEKSGLNSNIGFWGLAKKGLILLIILLGHRMDM
LIGSDVVMNAAIYFYLVNELISITENYGRLGLPLPDKIRGVIAQLKNSKQD
>lcl|NZ_CP026520.1_prot_WP_042228999.1_1622 [locus_tag=PC41400_RS08395] [protein=pirin family protein] [protein_id=WP_042228999.1] [location=1842727..1843428] [gbkey=CDS]
MIKIYPASSRYSTDHGWLVSHFSFSFADYYDPDNTRFGPMRVLNDDTVQGGRGFGAHPHREMEIVTIVLEGQLQHEDSSG
HRAVTTYGGVQRMTAGTGIIHSEMNPGQDDVTLLQMWFDPEERGLEPSYETTEFQVQALKNNLLPVVSKPKAGPGVAGIH
QDMSIYLSELDGGKSVTFEQPEGRRIFFMVLDGEAALNSGTILGKRDSARITETPSLTISSEAGARFMLIDLP
>lcl|NZ_CP026520.1_prot_WP_042228998.1_1623 [locus_tag=PC41400_RS08400] [protein=hypothetical protein] [protein_id=WP_042228998.1] [location=1843454..1843783] [gbkey=CDS]
MSEAHRLYVKHAVGSRTLLDTKELGGFLHLSKVPGGWRFEISGVDLDAAREIADFREELNLFYLEEGEGEERQKWWYYGH
KTPEIDYEANGRVLHITVDTRKGYSNRHV
>lcl|NZ_CP026520.1_prot_WP_042228996.1_1624 [locus_tag=PC41400_RS08405] [protein=hypothetical protein] [protein_id=WP_042228996.1] [location=1843928..1844194] [gbkey=CDS]
MSTLERYLGRKVEIIYMDRKGAISQRDIRPLAIAGDKVKAYCYAQQGLRFFRSGHILAWVPQPEEETSGKPEFYPRRKKT
PGNPRLYG
>lcl|NZ_CP026520.1_prot_WP_042228994.1_1625 [locus_tag=PC41400_RS08410] [protein=TIM barrel protein] [protein_id=WP_042228994.1] [location=1844306..1845124] [gbkey=CDS]
MKVSVGGFSFNQAFLNGTMDVFGYLETVRYRYGLSTVDLWNGQFADRSHPLLKLPDREYLKKIRQAMDEKEMTLVNIAID
GAHLWDPDADKREALHKNALEYLDASILLGAKTVRIDTGGRGQSTFSDEELDYVAGRYREYCQIAQENGFTIGPENHMGP
SLVPGEMKRLAEAVNHPSYGFLLHMHRWDQDREQGDAIVAPWTVHTHFDAQTAGADDAVETIRMLKKAGYDGYWGVEHHA
HKQPYLEVEWMLTTVKHLLLMSEDDNNGEEQV
>lcl|NZ_CP026520.1_prot_WP_042228992.1_1626 [locus_tag=PC41400_RS08415] [protein=Gfo/Idh/MocA family oxidoreductase] [protein_id=WP_042228992.1] [location=1845126..1846232] [gbkey=CDS]
MNAIRIGIIGVGQIGKIHLDQYKLIPDAEIVAVCDVNEAEARRVADLYGIPEVYTDYNELLKRDDLDAVDVCLHNNFHAP
ISIAAMKAGKHVYCEKPIAGTYADGRAMVDASREFGRLLHIQLGTLYRKETKAAKALIEGGELGRVFHARSTGYRRRNRP
FVDGYGTPAFTRKETAAGGALFDMGVYHISQMLYLLDLPKVERISGKLYQETDMDPGRRESSGFDVEELAIGLVRFEKGI
TMDIIESWAVHMNPFEGSSVLGSKGGLRLPVNGPDGNKLTHYSTRCDMDMDTTIDLDAADLRWHRLRADEEAYDSSQHHW
IAALQGRVKLLPTAEIALQTMLISEGMYLSDAQGREVSADEVENGLKA
>lcl|NZ_CP026520.1_prot_WP_042228991.1_1627 [locus_tag=PC41400_RS08420] [protein=MFS transporter] [protein_id=WP_042228991.1] [location=1846462..1847670] [gbkey=CDS]
MTTGAAERRSFTMFKLYILFLYGAFSLLFTYVPLLFLEHGLTRFQIGMLMAGGPFVSILANPFWGYWSDRMQNVRRTLII
LVICNLVVVQPVFHLHSFSWLYLVLLLYFFFQSPTFSQSNSLILSTIENTPYKFGTFRLWGSLGWAIAAVAAGPVLTAVG
IEQLWWLYSGVLMISLLFVFSIPQPKPAAASDALPGGGFKQLLGNRPFVMLLLLGVLISVPNSINSTFVSIYLTDMGGSA
TWIGWAAFLSSVFEAPVFLLLDRYLKRDSRTMFVCLALVSLLFSIRWLLMMLVDSPMQLIFIQLLHCITYGGYYYIGTNL
TGHLIPARMRASGQAAFALTWGGVSGIFAGFAGGWMYDTLGAGKMYGIGAGVSLIGMVGFLLIWRGMRQSAADQGTDAAA
GV
>lcl|NZ_CP026520.1_prot_WP_042228990.1_1628 [locus_tag=PC41400_RS08425] [protein=ROK family transcriptional regulator] [protein_id=WP_042228990.1] [location=1847810..1848946] [gbkey=CDS]
MNQIGNQQMLREINKSLLLNNIYEQGPISRVDLARKTRLSPTTVSVLMEEAIREGLVRETGTAGSGVGRRMTLLDINAAG
GCVLGVDLSNAPARCVLLDLRGNLLASQPLQPAQPLFGPEAMCSELPQLIRAFLEEQETPLATIRWMGVSIPGRIDEKRE
NVVNSTVLGLENFPLRTKLYEEFNIPMHLVNDLDAAGFAERFNGTAKGLQTIVYILIDYGIGAGLVLNNQIYRGRHDRAG
RIREFSRFSTANLSKRLLETYPDTFGDVRDEEPLVVRRFVDLGMQGAEPFAGELEEIIETIASYCGHVLQLLSPEQLILG
GWIGESEPFFTRLAERIQSREQASVPTPVHSPRWKEYGASIGAATIGLHRMFKSRTMS
>lcl|NZ_CP026520.1_prot_WP_042228988.1_1629 [gene=nagA] [locus_tag=PC41400_RS08430] [protein=N-acetylglucosamine-6-phosphate deacetylase] [protein_id=WP_042228988.1] [location=complement(1849027..1850223)] [gbkey=CDS]
MSRSTSAQERILIVNARIVTENTIVDNGALLAVDGRIERIYKAHELAELDLTGAVDAQGSWLLPGFIDVHVHGGYGADFM
DASAKSLETITRFHSTKGTTAMLATSVTASREALDAVITSTDAYIRGGMPYAQLIGVHLEGPFISPKWSGAQNPEFISLP
TSEWLEDWQTRFPGLVKMMTLAPEREGAMDVIQWLSDHGIVAAAGHTDADYETIQRAVEHGLSHAVHTFNAMTPLHHRNP
GTAGAVLTDPRIHAEIIADGVHVHPAGVQLLTQTKKQHNLIMITDAISAAGLEDGDYKLGGLDVIMKDGICRLKEGGSLA
GSTLTMIDAFKFMVNKIGVSVQQASEYASGNPARQMGVADRMGSIARGKQADFLLVSPSLDLERAWVLGRELSLVTEA
>lcl|NZ_CP026520.1_prot_WP_042228987.1_1630 [gene=nagB] [locus_tag=PC41400_RS08435] [protein=glucosamine-6-phosphate deaminase] [protein_id=WP_042228987.1] [location=complement(1850220..1850948)] [gbkey=CDS]
MIIHKFENQQELNEAGAGIITALVQTKPDAVLGLATGSTPIGIYEEIVKAYNKGRVSFKNVRTYNLDEYVGLPDGHAESY
RTYMNEHLFRHIDVQPERTRIPNGNASDPQEECARYNRELDEIKQVDLQILGLGHNGHIGFNEPADELHSGTHLVELDHK
TREANARFFPSINDVPTHALTMGVGTILKAKTILLVVRGADKADIVHRALTGPITTQCPASLLQTHPHLIVLLDSEAGRN
FV
>lcl|NZ_CP026520.1_prot_WP_042228985.1_1631 [locus_tag=PC41400_RS08440] [protein=MurR/RpiR family transcriptional regulator] [protein_id=WP_042228985.1] [location=complement(1850952..1851809)] [gbkey=CDS]
MSSILRAIEERKHGLNRQEQVLAIYILEHSSQVVRMGITELAEQSKTSPATISRFCKTFHFQGYTDFKFQLGAELATQPA
GQSYQDIVAGNPLEEIVSAMEANHLRSIADTTRLLDLRQVQSAVDALQSARQIDLYGVATSGVVALDFHQKLVRIGRYAT
AWSDPHMQITSASNLQPGDVAFAVSYSGDTPETIQALLCAKERGATTISLTKYGSNPLADLSDIKLFASTLEEGMRRGDM
ASRIAQLHVIDILFTSLVSDRFDEHVTRLERTYHMVRKYRSKGRE
>lcl|NZ_CP026520.1_prot_WP_042228984.1_1632 [locus_tag=PC41400_RS08445] [protein=HAD-IA family hydrolase] [protein_id=WP_042228984.1] [location=1852027..1852707] [gbkey=CDS]
MNQSPAPQAMLFDLDGTLFKTETLLLPAYHAAFDALRREGVYEGPTPSDERILGGLGMLLEEIWERVLPDVPAAVRSRAD
ELLLAEQMDRLLRGEGMLYDGVEETLQALHARGIQLFVASNGLEDYVREVIRMKGLAPLFTGLYSAGEYATKSKVDLVKL
LLEQHGLQSAWMVGDRSSDVEAGHRNGLKVIGCDYAGFRSERELDEADVRITAFPELLGLLPRVEN
>lcl|NZ_CP026520.1_prot_WP_042228983.1_1633 [locus_tag=PC41400_RS08450] [protein=glycosyltransferase family 39 protein] [protein_id=WP_042228983.1] [location=1853038..1854303] [gbkey=CDS]
MGASRLWKKMLIVFILALALGLRLFYILGEQYEPVEYDQKNYTTMAVQMLEKGIYGYNEAKSNTLVTPGFPVFLAAILAV
FGYTDIERTFMVVRVLQVFISLGAIWYIYKIGLRLFNRTTGLIAAFFASVYGTFIVVNSLILTEVIFLTSFMALIYYQVK
IVQENRTRDHLLAGLLLALAVLIRPNTLVVAPVPYIFLWFQHKKPYFKEISFGVLAFALGMLPWWIRNAITFHEIIFISK
GGAGNPLLGGTDPYYRGTLDWSKIDFSNQQGEAIKRIKEGLQTDPALWIRWFTVGKMKEMFLNKLYIGDYGSSVPAWYAP
WIPRFHKFLVYAGMGGSLIAFFIHKSYRYLITVFLILLGTQLMFIPEARYTIGMMPFLMLIISALLVQAVSYAAGMVRGR
NKTGSGVRKDVRSETDRSKAV
>lcl|NZ_CP026520.1_prot_WP_042228981.1_1634 [locus_tag=PC41400_RS08455] [protein=flotillin family protein] [protein_id=WP_042228981.1] [location=complement(1854504..1856072)] [gbkey=CDS]
MPNTDVLLIPGIIVGVILILGIAFWARYKTVSPDEAMIVTGSFLGSKNVSIDDSGRKMKIVRGGGAFILPIFQQAQFLSL
LSHKLDVMTPEVYTEQGVPVMTDAVAIIKIGGSVEDVATAAEQFLGKPTEALKSEAQEVLEGHLRAILGSMTVEEVYRNR
DRFAQEVQGVAARDLKKMGLQIVSFTIKDVRDKHGYLDALGKPRIAAVKRDADIAEAEANRDARIQKAKAEEEGQKAELL
RDTNIAEASKEKELKVAAFKKDQDMAKAEADQAYHIQEARSKQSVVEEQMRVELVRKEREIDLESKEILRREKQYDAEVK
KKADADRYSVVQAAEAEKSRKMLEAEALQFRIEAEAKAMAEQKRLDGLALADAERARGTAEADVIRLRGLAEAEAKQRLA
EAFEKFGEAAVLDIIVKMLPELAGKVAEPIKGIDKLTVVDTGHGEGAARISNYVTSLMATAPEMLKNVSGIDVEGLIKGL
TSRSAKPAAKPAPLTAELEPAPAAAAAVGAIAAAKEDETDQG
>lcl|NZ_CP026520.1_prot_WP_042228979.1_1635 [locus_tag=PC41400_RS08460] [protein=NfeD family protein] [protein_id=WP_042228979.1] [location=complement(1856094..1856630)] [gbkey=CDS]
MESWYWGLFIAGILFALATLIFGEVLGHLSGMLDSASGHHLPFLQPAVLVGGITTFGGAGILLSRYTELTSWQSGGLSLL
IAAVLSVLVYLVYVKPMENSENSVAFSLQELVGKIGEVTVPVPASGYGEVMVRTGVGVTNQIAESLDGRSIPSHLRVVVV
EVKNSVLLVSPLDSDPLN
>lcl|NZ_CP026520.1_prot_WP_042228978.1_1636 [locus_tag=PC41400_RS08465] [protein=glycosyltransferase family 2 protein] [protein_id=WP_042228978.1] [location=1856901..1857596] [gbkey=CDS]
MSRQSDVLIIIPAYNEEDAIEQTVDHLREHVEYDYLIVDDGSKDRTAAIIRERGYNGVRLPVNLGIGGAMQTGYRYAARH
GYKYAIQLDADGQHDARDIDKLVREIRESGNDMVIGSRFVEKSSYKSSLPRRIGIYYFYMLIRILTGNRIFDPTSGYRIV
NSKIIREFATYYPVDYPEVEILVSMARSKYRIKEISVEMKKRQGGVSSINPVRSVYYMLKVTMFSIIRRSF
>lcl|NZ_CP026520.1_prot_WP_042228976.1_1637 [locus_tag=PC41400_RS08470] [protein=DUF2304 domain-containing protein] [protein_id=WP_042228976.1] [location=1857611..1858000] [gbkey=CDS]
MIQAITIGVTVIFLFQILFYTSKHKLRDRHAFTWIMIALAGLVTGICIPLLNRFALWIGVSYMPSLIFLIVIVVTLSLLV
HQTILLSKQQEQIKTLAQEHAYLENEVMRLKEQSRNQDSYTGTERRETP
>lcl|NZ_CP026520.1_prot_WP_042228973.1_1638 [locus_tag=PC41400_RS08475] [protein=EamA family transporter] [protein_id=WP_042228973.1] [location=1857997..1858368] [gbkey=CDS]
MNPLHIGLILLNTLMLVSGQFLWKFGLSRKADPFESLQSIIHLMFSPFILGGLFIYGLATVLWLFILNKVDISIAYPMQS
IAYLITVIGAYYIFNEQMSLLKIAGCVVILIGVGMIGLSARYS
>lcl|NZ_CP026520.1_prot_WP_042228972.1_1639 [locus_tag=PC41400_RS08480] [protein=glycosyltransferase family 39 protein] [protein_id=WP_042228972.1] [location=1858407..1859666] [gbkey=CDS]
MPHRLRSIPVRGWLWMVVLLTTLLKLVVVFKYGSSYNLGSDDQSYLETARIWLEKGMFTYNDSSKPTVFITPALPAFIAL
FMSGLGTGYVLEQTIRVVQVLMVSFSMILLFKIGIRLFNERTALIAVVLTAVYPPMWNISFYILTEALFVLALMLLVHGA
LKAMDKPTAGNAVWFGLMWAFAVYVRPTIALWPGFFLLLLIYWRAIPWKKLAVCLAVSALVFVLCLMPWWVRNYEVSGGQ
FIPLTKSSGNPLLLGTFPYGPPPVAEQRTWHASSDLWVNDAFDTQWAKERIRTGFAKQPLYYLQWYTIGKFATFWGDVYY
WNAVNWVSLILPVGIHYTILGFGIRGIWLARGKRLTYLLTVLFAYMSLLHMIYLAHSRYSAPLMPFMALFAGYAILQAWE
RRGRTGAAAGRKHQKPSLP
>lcl|NZ_CP026520.1_prot_WP_042228971.1_1640 [locus_tag=PC41400_RS08485] [protein=YheC/YheD family protein] [protein_id=WP_042228971.1] [location=complement(1859692..1860819)] [gbkey=CDS]
MDVSAHKKQQPLCIGILTTPDPKRQFSGNRENFIDLIKAGYEQGALVYVVTTSDLKLSGKQVQAYCYQLASKKWNKQLMP
LPDVIYNRIPTRRLEIQSDVQRTIQACIRSKQVHLFNPFFFDKWTLFEWLDQSPEIRKYVPVTKQLSSQAVLESMLKRFP
FIYLKPVEGKAGKGIMRLERITKKSGAQTYKLTMQETADSHYELFTNLSDVWNRIAENKKDENYIIQQGILLSNFNKRPF
DLRALLQKTSKGTWSVTGIGARVAGKRSITTHVPRGGSIDDPARLLEKSFGPLKAKRILQRTRSTALLMANSVEQTSKQK
LGEMSMDLGVDLDGRIWFFEANSKPMKFDEPHIRKKSLTRIIGYSVYLANKNKTK
>lcl|NZ_CP026520.1_prot_WP_042228970.1_1641 [locus_tag=PC41400_RS08490] [protein=YheC/YheD family protein] [protein_id=WP_042228970.1] [location=complement(1860819..1862186)] [gbkey=CDS]
MSLTTCTVHVNSKQEKNILVSGELAKALRLSKNKTLQLQVGSKSITLPARYIKKSGLHLYLPQSVITSVKLPRIGPCLAA
SGSGRDIRLGPVIGILTNVSSTTAGPFGTRTPFIRSLMQPGERKSFHIAFSPKDVNWQQGTVNGYMLQPQGGWVRKTVPL
PDVVYNRLPSRKAEKLLSMEDFKQRFVRRNIPLFNWAFFDKWDVYRLLENDPAFKYVPESVLDPTAEQLQDLLTRHKFIY
LKPTAGSLGIGIYRITYSTDRGYFVRYRRNGKNVLLRFNKYEGLVKMLGINRGKLNSYVTQQGIRLIEIDGCPIDFRFHL
TKNGSNQWVVAAIGAKKAGKGSVTTHLRNGGQLMTAEQALKQVYGNRSDAVLTRMKETSIELAEAIETNYNHRLGELGFD
LGIDQNEKTWMFEANAKPGRSIFKHPGLKSQDKASLNYIYEHCVYLSRFRAGRDG
>lcl|NZ_CP026520.1_prot_WP_042228968.1_1642 [locus_tag=PC41400_RS08495] [protein=YheC/YheD family protein] [protein_id=WP_042228968.1] [location=complement(1862191..1863336)] [gbkey=CDS]
MDHTVNRCLGVLASPGKTAPAGGSESFFRRLSQNGAAEGIEVIVFSPDGISFEYETVDGFLYSPSDSAWREGTFPLPAVL
YDRCFCSNSAEQRLYRRQLNRLHRLPGIRFLGGGLRGKWDVQRLLAADPALAPHLPLTRRLTDPHQLEPWLELHGAAFLK
PEAGMHGKGTLHIYCSPGTDFSWSVRGRDRANRPVRQDFGTTLEMRRWVRDFIGKRRYLLQQYLPLTTPDGTAYDIRSLM
QKDGRGLWKLTGMVVRSGRPGSVTSNLHGGGIPCEAEAFLREHFPGGASGRLMKTLSVLSHRIALILESSHGRLAELGID
FGVCRDGRIHILEVNSKPGRAAFQSIDGGKTARSAASNPVRYAAYLLRTQREPAGATSLGG
>lcl|NZ_CP026520.1_prot_WP_042228967.1_1643 [locus_tag=PC41400_RS08500] [protein=YheC/YheD family protein] [protein_id=WP_042228967.1] [location=complement(1863358..1864743)] [gbkey=CDS]
MTRIFHVRSLDPALYEDERTVLLSPEMIRACRLAPDSAVKLRFGSASCEAVVRPAAEPGASPDEELCLHPALRRRLGIHG
GCRLSLVYQPSAATLYAGPLVGVLLSRLYSSQEMPFGSMTTFCRELTKAGEEAGVSLFFFSPEEIHTFPDLGGLYYDEKW
KKGIFPVPHVIYNRLTSRKYENKPEVQHFLKAVQLRHQTHIFNDKYLDKNAVFSALGSETALQNCLPESHPFSNYHMLKA
MSARHRTLFIKPASGSLGKGIFRLHVREDGSCEVRFSALSGTVVLPYPDLFEAYFHMAGKIRTQAYQIQEGLDLLTIGGR
PVDFRALVQRDGGGEWVLSSIVARIAAEQHFVSNLARGGTMTAVPEALAGTSLASRKGSIQRTLKSTALIVAQAISRNVK
GTFGELGVDLALDTAGQVKLLEVNSKPSREEFSLPSPAEETLVRPSVRRLLDYAVYLAGFM
>lcl|NZ_CP026520.1_prot_WP_042228966.1_1644 [locus_tag=PC41400_RS08505] [protein=extracellular solute-binding protein] [protein_id=WP_042228966.1] [location=1865205..1866569] [gbkey=CDS]
MKRNKWLNMGISLMMVTAIAAGCSKTNTTTTPAATPGASSNANSGEAGGKKLTGEFEIQYFVGGYGDTWWKAAIDEFQKA
NPDLKIKQSAGAKINDQMKPRWIQGNPPDVVYIDGAGANARTMYQDGQLMDITDWLKTAKNADGKNIMDMFISKPEEYEA
GKYYDVPLVFGSWGTFWDRAEFKKNNWEEPKDWESFLAVGDKIKASGKMSVYIHTGVYPYYINGAFLDSATVVANNYDTS
IIKKINNMEEGVFKSEPVMKALDKIIEMRDKGFIDKGSVALNHTDSQTQFLQHKAAFIPNGLWVENEMKKDVPQGFDFGF
IPSVTQEKGGKNVAIPTTATMGIAKKAKNPEAAKAFIQFIFTEKQAGKWAEATGALLNVKTNLDNTSASNLAKTAIKSYS
SPETVVAPEPVIPEDLKKVRNDSTLALTQGTITKEEWAKRMEDEAAKQRAKLKK
>lcl|NZ_CP026520.1_prot_WP_042228965.1_1645 [locus_tag=PC41400_RS08510] [protein=sugar ABC transporter permease] [protein_id=WP_042228965.1] [location=1866647..1867600] [gbkey=CDS]
MKGTPVTNPAGVITAKKPKSGRSKVRRNLFIATFILPTLLLYLVFTVYPMLKGIYLSLFDWSGGSMDYNFIGLDNFKEMF
KDPIIPKALLNDYFLVAGKVIGFMVLALFFAVALTRLKIKGSGFYRAIFFLPNVISVVVVGVLWRYVYNPNLGFLNSLIG
KIKGEPFIFPWLGDAATAMWSLLPPSIWAGVGFCTLLLIAGILSIPDSLYEAAGLDGATQWQQFWHVTLPLTWEQIQTSV
IWIVMTTLNGSFIIVSIMTEGGPDNATQVLGSYLYQQGFRQYHLSYAAAIGVLTLVVSLVTTVALQRIMKREPLEIS
>lcl|NZ_CP026520.1_prot_WP_042228963.1_1646 [locus_tag=PC41400_RS08515] [protein=carbohydrate ABC transporter permease] [protein_id=WP_042228963.1] [location=1867614..1868459] [gbkey=CDS]
MSDGLKNPFWRIVVNLLLIVWAIMVLYPLVWTVLGSLKDNQQFFQGNPWDLPQFPLLFSNFSYVWDKYQFGGYFMNSIIV
TVVSVLAGLVLSSMTAYILARFAFKGSGVLYYVYIASMMIPMFLGMIPLFFLMSDMGLTNSLLGLIIVYTVSTIPFSVFV
LVGFFKTLPSEIEEAAMMDGASYYGIFFRIMMPLAKPGLISVSIINVLNTWNEYILGVVLNSEPSKYTLPVGIAVMQGEM
QYRTEWGPLFAALLISMVPVMIFYLIFQRQIASGITAGAVK
>lcl|NZ_CP026520.1_prot_WP_042228959.1_1647 [locus_tag=PC41400_RS08520] [protein=PTS transporter subunit EIIC] [protein_id=WP_042228959.1] [location=1868503..1869936] [gbkey=CDS]
MLHTLQKIGRSLMIPIAVLPAAAILQRLGAFTFQDPFLIFISKIFSTGGNAIVDNLPLLFAIGIAIGMTSGQGVAALAAT
VGYIVFVRALATFESVPVEQTTPGAHLDMGVLGGMMTGLITVMLFNRFHNIKLPDYLMFFGGRRFVPIITSLVMAGMGGI
MGLIWPPVQDVIRDAGLWIVDAGGIGVFMYGFLNRLLLVTGLHHILNSIVWFQVGEFMTNTGTYVHGDMTRYNAGDPSAG
MFMSGFFPTMMFGLPAAALAMIKCALPEKRKKVAALLYSAALTSMLTGVTEPLEFSFMFAAPLLYGVHAVLNGLAMLFMH
LFNIKLGFSFSGGFIDLLLNWKLKEFPFAFLIIGVVYFVVYYGLFLFLIKVFKLKTPGREEDFPGLTAAIVSSALPPEEE
RITAIVNSVGGIDNIVKVDSCITRLRLWLKEAEEVEEEKLKELGAFGIMNMGKGHVHVIFGTDSELIKESIAKRLAA
>lcl|NZ_CP026520.1_prot_WP_042228957.1_1648 [locus_tag=PC41400_RS08525] [protein=PRD domain-containing protein] [protein_id=WP_042228957.1] [location=1869953..1870789] [gbkey=CDS]
MIQRKIIHVLSHNVVMTKLSTGKNSIAFGKGIGFKKQPGMIIEDADISQEFLLHTSEVLEHYEHILHAVDVKIIGITEEV
IAYAQGHLEGEFSETIHAALVDHINFAVERCKRGVHILNPFAYEIKHLYEEEYRIASKAIDFLNQHLDVKLPDDEIAFLA
MHFHTARTKVEKGTALAVVRLVAQVIERAKELGLDFDNSFSTVRFISHLKALIDRIMEQKTIRNPLLGKISEEYPLAFQK
AVLLAELLKANLQKPIPMDEIGFLALHLERQMQRSGMD
>lcl|NZ_CP026520.1_prot_WP_042228955.1_1649 [locus_tag=PC41400_RS08530] [protein=acryloyl-CoA reductase] [protein_id=WP_042228955.1] [location=1870900..1871895] [gbkey=CDS]
MTDSAFRALQVDKTGQGVSLGIREMTVEELPEGELLVKVAYSGVNFKDGLASMPESKTVRSYPMVLGIDLSGTVVSSADP
RYKEGDEILVTGYALGTGHFGGYSELARVPADWAVPVPEGLTPKDVMALGTAGFTAALSIQRLEDNGLRPGQGPVLVTGA
TGGVGTAAVSMLAGLGYEVAASTGKSSEHAYLRGLGASEVLPREALAAEDGKPLRREQWSAAVDPVGGASLPYVLGTVRY
GGSVALSGLTGGGAFPATVYPFILRGVNLLGIDSVYCPEGLRRKLWERLAGDLKPRSLGDMIQREITLEEVPAALGDILA
GRMRGRVVVRL
>lcl|NZ_CP026520.1_prot_WP_042228953.1_1650 [locus_tag=PC41400_RS08535] [protein=ABC transporter substrate-binding protein] [protein_id=WP_042228953.1] [location=complement(1872168..1874096)] [gbkey=CDS]
MQLYDYYLQLRVSQPHAREGEPVPITLDLLAGHWFCTVRNVKLLLKKMAQLDWIRWQPGCGRGRSSLLTFVIPAPDLISR
TAMDLTRKGDFKAAVELIHTYGRHYALKDSFLDWMSGYFGFTAASSEGAVLDTLRLPLFRHPVSLDPAHAIYATDCQLIS
HVFSTLLELDPATGNLLPQLAHYWECSADGTEWTFYLRKKVYFHHGREMDAGDVKYSLERLADSATGSSWSWFIQDVVSL
TPRTPYILRIGLKRPNHLLPFFLTFPAASIVPEDIVRDSPSLFGRSPIGTGPFRMEFYDKYICRLHAFDGYYGVRPHLDV
IEMWILPEQLASYRPKGKDKLIVSHGDVPMPAETYKDQGFLAKEDHVFDGSTVLTFNTRRGPCRSLLLRQAIDRLIDREA
MARELGYNRIGPSSGIMRDFQLKQGSRLPEVPDLNAGYARDSTSLSVDELIRTSGYDGETLRLLAYAQHGEDARWIAGKL
TEAGLSVKVETMPIAAFTDGDYVHQADMCMFMPVSSSPLGFLGYLLQTTNYIHRHMSSALQSRIQNAVGQIQRESSLERQ
AAELLKIEDMLHAETALLFLLKKTFQVAAHSSIGGVRADRQGWVDYRHLFLRERGGEEALDEGDRTVDGSGLRVPDSADE
RP
>lcl|NZ_CP026520.1_prot_WP_164977113.1_1651 [locus_tag=PC41400_RS29700] [protein=hypothetical protein] [protein_id=WP_164977113.1] [location=1874226..1874393] [gbkey=CDS]
MYPQPYYLEKEMEQHNRDLDRRLRDYTKFGRFLHSKLMNSKNGTKNTSTTAKSAV
>lcl|NZ_CP026520.1_prot_WP_042228952.1_1652 [locus_tag=PC41400_RS08540] [protein=hypothetical protein] [protein_id=WP_042228952.1] [location=1874517..1874792] [gbkey=CDS]
MKGFKNVIFMSLALGMLLYSVPQLDIGGGFTLPTVFGALWICFALLIIAAHLHELLGVDEETRQEIVRVKRMKRWRMEQR
LQGKKILGLRK
>lcl|NZ_CP026520.1_prot_WP_009674567.1_1653 [locus_tag=PC41400_RS08545] [protein=DRTGG domain-containing protein] [protein_id=WP_009674567.1] [location=1874993..1876330] [gbkey=CDS]
MDETGTESLTKHEQILRHIESLKIGSKISVRKIADQLEVSEGTAYRAIKEAENQGLVSTKERVGTVRVGKRERRNIDKLT
FEEVVNIVDGEVLGGMNGLEKTLHKFVIGAMELDAMMRYVEAGSLLIVGNREEVHRLALEQGSGVLITGGFHTSPEMTKL
ADKLELPIISSSYDSFTVASMINRAIYDRLIKKKILLVEDILTTNPHVFALKANSSVKEWRALVEETGHSRFPVTDEWNR
VIGLVAPKDMLGADAGQTVDRLMTRNPLTVSPQTSVASAAHMMVWEGIELLPVVDGNRKMLGVINRNDVLKAMQYIQKQP
QNGETFEDLIWGRFEEFRDEQGNVVFRGTITPQMTNHLGTVSEGVLTTLMTEAAYRVVQTVKKGDLVMDNMSTYFIKPVQ
IESEIQIRPNILEVSRKFGKIDVQIYHAGTLVCRAMLTAQFIDGV
>lcl|NZ_CP026520.1_prot_WP_042228950.1_1654 [locus_tag=PC41400_RS08550] [protein=YtpI family protein] [protein_id=WP_042228950.1] [location=complement(1876331..1876627)] [gbkey=CDS]
MLETVQIVLSTLIFITLLFSVVFSFRSRRKTDPKQRGTNAALMNISMGLMLILIAVTQLFFLSDTPVRRIFGTVCLLLGL
FNLFAGIRNYLYFSRMSR
>lcl|NZ_CP026520.1_prot_WP_009674608.1_1655 [locus_tag=PC41400_RS29705] [protein=hypothetical protein] [protein_id=WP_009674608.1] [location=complement(1876794..1876937)] [gbkey=CDS]
MTQYQFDNFFEDLLKDTLIEKFETVEVPRLQLESSWEKLQQKLEISN
>lcl|NZ_CP026520.1_prot_WP_164977114.1_1656 [locus_tag=PC41400_RS29710] [protein=hypothetical protein] [protein_id=WP_164977114.1] [location=complement(1877290..1877457)] [gbkey=CDS]
MQTAVLWSRELKQAVKEAVSSKAAPVSSASLSELEVESLQKAVSSEKPEVRYMMW
>lcl|NZ_CP026520.1_prot_WP_042228949.1_1657 [locus_tag=PC41400_RS08555] [protein=ATP-binding protein] [protein_id=WP_042228949.1] [location=1877539..1879866] [gbkey=CDS]
MNSKKKWMILFFIGIIFLQFWCLRVAYRYPYLGIIVDQNPKQEWVITQLERVSIGSKIGFKPDDKILSVDGKRPEEHLTV
KKYYSIEQADVITVLRGDKELTFTTRDHATSMTVDILSLLGEIIAFVFAFLIYSRLSKSRSALMLSAVFLNIGLIFMCLG
ASTRTDILAKGMTTSLVACLPFLFLQFLIMFFQERSNFIFSTRYMKYVYAAIGFSSLLQLSMFFPEANFKFYKIIYYGTM
SFFIAGFIMNIYLLAHLYKKYRKSNDYLFTIIKTIIFCLGLSFCTLAFLNFIPRILFQTELAFSYYSGWFILFFPVTFGY
LIMSKQLYDLDIILRRVFYTAVISVLPSAVLVAVNYFIFQKDATLKHMVISFTTTMVVLTFVTYSLEYFATRLEKIMFPR
KYYLQQALKKIASALGSITNFRELRDIVLVDIVDILQVYGGAIVIQYASGTLEIISEGNIDEQDVKRRFLTGDWFDSEYT
RTLIKGHEEYVSYLVMAKKKTNTFMNREETQWLSLIISYLSVSLENLYLIRKLNMKLLELASELPNQEGSEEYVWFRKTM
FDLQEKERTRIATDLHDTTMQDIFFVQRKLKNLQKQMLNPDEHRQLKDILNHLELINMNLRQCCFELNPYLLKNIGLVRA
LQNLVDTESSLGDYEIQFVTEGKTTAMENLDYEAKRHIFRVIQELITNAKKHSQASRLFIRLVASGNEVTIHYEDDGIGF
KPNSSSGKTVGRSGMGLEQMKGRVLHLNGQFTMESMTGKGVEVRIQIPIKEGIPT
>lcl|NZ_CP026520.1_prot_WP_042228946.1_1658 [locus_tag=PC41400_RS08560] [protein=response regulator transcription factor] [protein_id=WP_042228946.1] [location=1879863..1880486] [gbkey=CDS]
MNKIQHILIVDDHPAMAYGTKFILEQEENIQVVGVATTGEMGVEMMVELNPSIIFLDFHLPDLTGMEVAVKMRELNKDVQ
IVIFTGIDYMPILNNLLELGVCGIMSKDSSEDQIRNMMRALKEGQSVIPISLLHQMRLSHPDDDLSSLTDEEINIMNQIV
KGATNEQIADEIHMSKRSVDNYVRKIYDKFGVKSRAQAIEKFLQIKR
>lcl|NZ_CP026520.1_prot_WP_042228944.1_1659 [locus_tag=PC41400_RS08565] [protein=polyprenyl synthetase family protein] [protein_id=WP_042228944.1] [location=1880511..1881449] [gbkey=CDS]
MKAEVWSDIQDLIETYFTVPSLKELLLKSVADKKEEHSIWGELTWSVHDLLGGSNPAIHRAAALTELMLLALDIMDDIQD
QDNTEKPWMTVPQEQALNVYTGLVTASVSELGALRRQYPAAPLPDPAKVCEFITEAINGQHLDVTNAVQTEEDYVGMVQR
KSGTLVQLALYMGYACTDANADEDTVGKLNRLAVYMGLMAQLDNDLSDLLRFDYKNDLVQKKKTIPVLFLLDEEDTEFPV
IQQYYNGEIDQPQFLKHKPQCVQYVVDSGCIEYTRTIQNLYLHHADELFAEIPAASANKEAFREIALARFEL
>lcl|NZ_CP026520.1_prot_WP_241688885.1_1660 [locus_tag=PC41400_RS08570] [protein=hypothetical protein] [protein_id=WP_241688885.1] [location=complement(1881491..1882036)] [gbkey=CDS]
MPSFDRYPRFLAGIGLFLAGGIVGSAVFLSLYHHTYNVLVIENWNLKKENNGLQLDIETLNKFKNKQGLITRVTVHLYND
IQHPLESHVQKALEDRVKRDLKIVEGQKLGEIRDSPQLYERLIANKTYYGVIDKDYIVNVKAMALVQSELTVWITADEPK
KQALPDKGTIVPDHDAFNGSY
>lcl|NZ_CP026520.1_prot_WP_009674587.1_1661 [locus_tag=PC41400_RS08575] [protein=YtrH family sporulation protein] [protein_id=WP_009674587.1] [location=complement(1882039..1882371)] [gbkey=CDS]
MTGFVTSAILNFFVAFGVVMGGSMLAGIGSVLSLQPPTYKMLLIAENIKIWAMVAAIGGTIDPIRMIQDNFQDGQLSPAF
KQILLIISAYIGAHMGTSLIQLICKGGGQS
>lcl|NZ_CP026520.1_prot_WP_053228690.1_1662 [gene=hisC] [locus_tag=PC41400_RS08580] [protein=histidinol-phosphate transaminase] [protein_id=WP_053228690.1] [location=1882600..1883694] [gbkey=CDS]
MYAKAHIHSMAPYIPGFQPEAEQQVIKLNSNENPYPPSPDVRKALAGFDYALLRRYPNATAEPVREVLAEIYGVNRNQVF
CGNGSDEIISLLFKAFLEEGSTIALPYPTYSLYRTAADIHRVHCEFVPTRDDFSIDTEALLAVPSQAIILVNPNAPTGLL
VSVDVIEELAGRYNGLLIVDEAYIDFADGEASALRLIGRCPNLLILRTLSKSYSLCGIRFGYAFADEALIEALDKCKDSY
NVSMLTQTIAAAALRDRSYFDETVGRIRASRGSLGGSLRRLGFAVPDSETNFLLCTPPAGSGLSARRITEWLAERHVYIR
HFDSPLLEDKIRISIGTEEENSKLLTLLEDLLNRPEDETVPSGD
>lcl|NZ_CP026520.1_prot_WP_042228940.1_1663 [locus_tag=PC41400_RS08585] [protein=pyridoxal phosphate-dependent aminotransferase] [protein_id=WP_042228940.1] [location=1883742..1884941] [gbkey=CDS]
MTSPAKGAISEKLAGQLEKGSWIRKLFDEGKRLIDIYGADEVYDFSLGNPIVEPPEAFLKALADTVNSGELGKHQYIPNQ
GLPPARQKVADYLNGRYGGSVKPEHIVMTVGAGGGLNVVLKSILNPGDEVILLTPYFVEYDYYADNHGGVAVHVPLADDF
SVDTERVAGAITERTRAIIINTPGNPTGTLLTQDNLNELGKVLAEGEKKHGGLIYMVYDDPYSQLVYDAVPPNPFAAYGR
TILVSSFSKDLGIAGERLGYVAVDASADDAQLLAQAFVFSNRTLGFVNAPVMMQRVITRMDTLAVEQQEYRKRRDLMVGV
LQEAGYEFTVPQGGFFIFPKTPIADDVEFCLKAVQEHRLLIVPGSGFGRGGHFRLSYSVPIKQIERSGKVFKALLDSYR
>lcl|NZ_CP026520.1_prot_WP_129112461.1_1664 [locus_tag=PC41400_RS08590] [protein=DNA polymerase III subunit alpha] [protein_id=WP_129112461.1] [location=1885317..1889540] [gbkey=CDS]
MNSFVHLHVHSEYSLLDGAARIQDLVETASQLGMKSLALTDHGVMYGAVPFYKACRAKGIKPILGCEAYFTEGTIRTKGT
RQEQPIYHLILLAKNMQGYQNLMKLCSVGHLEGFHYKPRIDHDHLARYSEGLVCLSACLGGEVSQHLLHDRKEQAKAAAE
RYRSIFGEDFYLEIQDHGSIEQKKVMQGMIELSRETGIPLVATNDVHYVREQDHVVQDVLLCIGTGKTVDDPQRLKFETS
QLYLKSEDDMRKLFAHVPEALDNTLLVAEKCELVLNFDQSILPKFKPIPEGLSAGEYLKELCTEGLARRYENQPGWNDLQ
GEYRQTLETRLSYELGVIESMGFSDYFLIVWDFIRFSHEHGIMTGPGRGSSAGSLVAYVLRITDVDPIRYKLLFERFLNQ
ERITMPDIDIDFNDERRDEVIDYVVAKYGQEHVAQIITFGTMAAKAAVRDVGRVLNVPYNEVDRAAKMIPGQLGMTLEAA
LKVNPDLKDLAEKQPKTAELLNMAKRVEGMPRHASTHAAGVVISREPLTHYVPLQEGTGQTPLTQYSMEHLEAIGLLKMD
FLGLRTLSIIERTLKSIEEQEGIKLDLAKLDMDNDPLTYELLGKGETTGIFQLESPGIRKVLRDLKPSGFEDIISVNALY
RPGPMEFIPRYIECKHGLTEPQIPHEALVPILKDTYGIIVYQEQIMQIASVMAGFSLGEADLLRRAVSKKKREVLDEQRA
HFVEGSVGLGYKEEEANGVYDMIVRFADYGYPRAHATAYGVLAFQTAYLKAHYPRHFMASMLTAVVGNHRKTAEYVDECR
RMKLEVLPPDVNESGVNFTPVAPKATLQVEEGTDEGHAGTDNGDKDREEAAVTAESIRSKGEDRIIGDSGESAKNDAGAS
PGGKGAIRFGLAAIKNVGTNAIEAILKERREKPYESLLDFCRRVDLRVCNKRVIESLIQGGALDGFPGHRAQFVAMLDDT
VEAAVKWRKERDDLQLHLFGFVEEPNWEVEIPQVAKLSRTQQLEYERELLGLYISGHPLDDHQELLQELEADPLHYLQDY
ADNSEVIVAGMVVSSKTIVTKKGQPMAFMDLEDRIARVEVVLFPEAWKKHAPLVQKGSVVLVKAKLQLQDEDPPKLLAEA
VAAPDDPAVRRWQRRPLRPGGQSASGGSAPRGAGTPPQGGRAGERGAAAGNARSGQPGEGPGAGSGANVNPGAGTRPGGE
AGAQAGVSGGARSAEGGGRADAVSGRAQAPGRAAGSARGSSVVRETRRGAAGYGVGEPGPAAGEARSGQDGGPRPGGGAS
AHAGSTGANAGQPGSSGRAADRGSGAPQRSSAAKPADESGRPAAQGTPAGTSRQRVYVKIAAHMEEPHNLVKLQELLQLH
AGPLEVALFYERTQRLLALSGQYRVKPSPELFRSIEQLMGKDCVKVK
>lcl|NZ_CP026520.1_prot_PC41400_RS08595_1665 [locus_tag=PC41400_RS08595] [protein=phosphatidylglycerophosphatase A] [pseudo=true] [location=1889634..1890184] [gbkey=CDS]
MSAELVLQMLRERGVTAESIAEIVLQLQKPYNKALTTEVCLESVITVLAKREVQYTLMTGIALDELAERKLLPEPLQSIL
EADEPLYGVDETLALGIVHVYGMIGLTSFGSWISRKSALSAILTTIGRRFTCFSTISWPGWPQRHPPESLIKTNPFRLIS
RNFAKPA*PARPA*SVRIRKSLH
>lcl|NZ_CP026520.1_prot_WP_156972832.1_1666 [locus_tag=PC41400_RS29715] [protein=hypothetical protein] [protein_id=WP_156972832.1] [location=complement(1890287..1890442)] [gbkey=CDS]
MNLHRLRGMEKEGLNKPHCRVHSPDTKAFPSVKYSLDRISRQKKLPTPTGW
>lcl|NZ_CP026520.1_prot_WP_009676703.1_1667 [locus_tag=PC41400_RS08600] [protein=hypothetical protein] [protein_id=WP_009676703.1] [location=1890619..1890807] [gbkey=CDS]
MWTVIYIAPTAKIAERIKQRLSEEGFLVQIRAISMTKNQFEIVVPEGEVEEVQEVLNTILHR
>lcl|NZ_CP026520.1_prot_WP_042231163.1_1668 [gene=accD] [locus_tag=PC41400_RS08605] [protein=acetyl-CoA carboxylase, carboxyltransferase subunit beta] [protein_id=WP_042231163.1] [location=1890891..1891757] [gbkey=CDS]
MLKDLFQKRKYATIPSEKAKRDIPEGLMNKCPRCGTIQYSKELEKNLKVCTSCNYHYKLNAMERMTMTLDEGRLFEYDEN
MVSEDPLGFPDYENKITKAKDKTGLKDAVVTGEGTIGGYPVVFGAMSFDFMGGSLGSVVGEKLTRAIESAIQKKYPLILF
STSGGARMQESILSLMQMAKTSAAIQKLDEEGGLFVSVITDPTFGGVSASFAMLGDIIIAEPAASFGFTGRRVIEQTIRQ
KLPDNFQTAEFNLLRGQVDKVVARKDMRTTLIKLLDMHVMKGDANYGG
>lcl|NZ_CP026520.1_prot_WP_042231162.1_1669 [gene=accA] [locus_tag=PC41400_RS08610] [protein=acetyl-CoA carboxylase carboxyl transferase subunit alpha] [protein_id=WP_042231162.1] [location=1891747..1892721] [gbkey=CDS]
MAGELSFEQPLTELRSKIDELKRFGEEKKIDFTDEIRRLEERYAELQQELYSTMSASEKMQLARHPQRPTTIDYIERIFT
DFVELHGDRLYGDDLAIIGGVAKLDGLPVTVVGHQKGKDTKDNIARNFGSPHPEGFRKALRLMKQADKFGRPVITFIDTK
GAFPGNAAEERGQGEAIARNLLVMAGLRVPVVCVVIGEGGSGGALALGVGNRVLMLENAIYSVISPEGAASILYNDASKS
LQAAEAMKITAKDIAELGVIDSIVSEPQGGAHRDLELQSANIKAAVLEHMQPLLAMSEEELKEDRYLKFRQIGRYTFTQE
GNHA
>lcl|NZ_CP026520.1_prot_WP_042231160.1_1670 [gene=pyk] [locus_tag=PC41400_RS08615] [protein=pyruvate kinase] [protein_id=WP_042231160.1] [location=1892714..1894474] [gbkey=CDS]
MRKTKIVCTIGPASESKENLKKLVTAGMNVMRLNFSHGDFEEHGGRIKNLRAACEELGGRNVSILLDTKGPEIRLGKLKE
EPIELAAGEMITLTTEEILGDRERVSVTYEGLAEDVKPGDTILIDDGLIGLTVEAVQGNDIKCLIVNSGPIKSKKGVNVP
GVSISLPGITEKDANDIVFGIEQGIDFIAASFVRKASDVLEIRELLERHNASHIQIISKIENQEGVDNLDEILEVSDGLM
VARGDLGVEIPPEDVPVVQKQMIKKCNLVGKPVITATMMLDSMQRNPRPTRAEASDVANAVFDGSDAVMLSGETAAGKYP
VEAVETMSRIALRAEAALEYREIFLRQAHALQTTVTESISQSVANAALELDAKAIVTATESGYTARMVSKYRPKAPIVAV
TRNPQVMRRLNLVWGVQPVLLHGDAQTTDELFEQAVDGSIREGYVDLGDIVVITAGVPIGSSGTTNLMKVHQVGEMIAKG
TGIGTQHAKGVVVTAETAEEAVSKMKDGAILVTNMTDKDYMPAIERAAGLITDVGGITSHAAVVALSLGIPVIVGVDRAL
DLIKDDMEVTLFPEVGIVYSGRARVL
>lcl|NZ_CP026520.1_prot_WP_042231159.1_1671 [locus_tag=PC41400_RS08620] [protein=thioesterase family protein] [protein_id=WP_042231159.1] [location=1894593..1895024] [gbkey=CDS]
MSNETTKEREWFPYGVRVRYQETDQMGVVYHANYLTWLEIARTEYIRDKGFPYTRMEELGLLLPVVGAELKYVRPARYDD
EITILTRVTSYTTVKLEFEYEVRRGEELLVTGLTRHAWVNLSWKPVRIDKEVPELYEWLQRGG
>lcl|NZ_CP026520.1_prot_WP_042231158.1_1672 [locus_tag=PC41400_RS08625] [protein=FxsA family protein] [protein_id=WP_042231158.1] [location=1895028..1895423] [gbkey=CDS]
MLRIALAVFIIVPVLEIACIIGMAHLIGGWMTFGLILLTGLLGALIVRKEASRFLPYARNQLSMGQNPTDSVLDSVCIFV
GGLLLIVPGFLTDLAGLLLVLPFTRSLFKMGLLVLLQKLMRNGRFMFINRR
>lcl|NZ_CP026520.1_prot_WP_042231157.1_1673 [gene=ytvI] [locus_tag=PC41400_RS08630] [protein=sporulation integral membrane protein YtvI] [protein_id=WP_042231157.1] [location=complement(1895502..1896635)] [gbkey=CDS]
MRVFNPYLLHKIFRGALVLLCLVAGWLLLRYVSPLIYPFIIGWLLAYVLNPIVNMLQRKAKMPRWLSATLVILLLIAITV
TAMTLLVTKVIVELGTLAENLQSMINSWKDQAREFLNSAWFQGIVGQLNDFMRANNLDLSSNLSSVTGTIADFSSNALKY
IVNSLIHIIQSLPNIATVSIIVLMATFFISKDWYRLVMRYTMLVPETVRKATRLISSDLQKALFGFMRAQLILISLTTVA
VIIGLIILRVDFAVTIGLLIGLCDLMPYLGTGAVLVPWIIYCFFKGNLFLGIGLSVLYGLIVVVRQIMEPKVLASSVGLD
PLAALIALFVGLKLFGVVGLILGPVLMVVLTAVYKAKVFHDIARYIKNGPAANTEVR
>lcl|NZ_CP026520.1_prot_WP_042231155.1_1674 [gene=ppk1] [locus_tag=PC41400_RS08635] [protein=polyphosphate kinase 1] [protein_id=WP_042231155.1] [location=1897107..1899215] [gbkey=CDS]
MENKQKTEGVKSNDCYKYVNRDLSWIEFNRRVLEEAQDPTAPLLERAKFLSIVSSNLDEFMSVRVAGILDQMKAGYHKKD
FTGYTPSGLLKRLLKRANRMVADQYKAFRDLTRQLAREGIFFPSYDELNSTQQKEMDTYFHEIVFPVLTPMAVDQSRPFP
LVHTQSNYLAVVLESCVPTDLRTYFAIVRVPHNLGRLVRVPVRTNSKKLEFIRMEELIKHHIESLFSGYKPIAVNGFRLT
RNADLTLNEEGAEDLLEEIEKELKQRRWGAPVRLEVEKGIHPYALELLRDEFEIGEGCFEIDGPIDLTYLIKQLPQLVGG
RAHLYYPVLEQVYAEEFEHTDDFFEVIRERDVLLYHPYECFDTVADFILQASDDPYVLAIKMTLYRVSGNSPLIGALARA
AESGKQVTVVVELKARFDEERNIAWARKLEKAGCHVVYGLVGLKTHAKITLVVRQEQESLRRYVHVGTGNYNDNTAKLYT
DVGLFTSHQVIGEDASALFNEMTGYSSPHDWQAFIVAPADMKTRLIELIEREGRHARAGKPARIVAKMNSLSNQEVVDAL
YEASQAGVKIDLIVRGVCVLRPGVADLSENITVRSIVDRFLEHSRLYWFENGGSPDVYLSSADWMTRNLTRRIELMCPVR
DEELQQSVIHLLELCLQDNVKARFLQPGGNYIRAAENGPKVRSQFDAMNINRWKVRKREYTR
>lcl|NZ_CP026520.1_prot_WP_042231154.1_1675 [locus_tag=PC41400_RS08640] [protein=Ppx/GppA family phosphatase] [protein_id=WP_042231154.1] [location=complement(1899175..1900710)] [gbkey=CDS]
MNSSRYAIIDIGSNSVRLVIYEVTPPGAYRILHETKETARLSERIGEDGAMTHDGILSVVPLLRRFKRICGAYSVDSVKA
AATAAIRNAANAEDIARELAERSGIVIDILSGEEEARLGFQGVVSTLDVRDGFIVDIGGGSTEITLFLDRKRVNSVSFPF
GAVTAAREYTPGGIVRPERIESLKALVRDTIDLHPWIREHAQLPLISLGGAGRALAKLNQKRVDYPLPVMHNYELEENDL
QSYMDLLPSLPVEKRKKLDGLSKGRADIIVAGVVLLHSVYRQTSASRIIVSSAGLRDGLFYETAMPERLPITDVLQFSVE
NMLKLHTPLPHDHLRRAAAHAAALYDVLADSAPELQPRHRSLLHAAALLQRIGYTVSSYQHWRHTFHLITESRIDGLTHR
EIVLCALIATFKNKNRTRQQALPYASVLDRSDEDIAAKLGSLLRLASSLDYGENGNIAELDVRFVPGELHLHLLCKEQDP
SEIGTIEETAEETGKYWSFTAYIHASALSTD
>lcl|NZ_CP026520.1_prot_WP_042231153.1_1676 [gene=citZ] [locus_tag=PC41400_RS08645] [protein=citrate synthase] [protein_id=WP_042231153.1] [location=1900985..1902097] [gbkey=CDS]
MTATKGLEGIVATTSSISSIVDGVLTYRGLDIDDLAENASFEEVIYLLWFGKLPNQAELDQLRSDLNASSSIPDQVLEQI
KLYPSDTNAMAALRTAVSALALYDKEANEMSRESNVRKAIKLQAQIPTIIAAFARIRAGKEPLASKGYPSLAANFLYMLT
GEEPDETAITALDKALVLHADHELNASTFAARVTVATLTDIYSGITSAIGTLKGPLHGGANEAVMAMLEDIGEVDNVVPY
INKMLENKEKVMGFGHRVYKNGDPRAKHLQKMSQQLTESTGLGKWYQMSIQIDELVTSLKGLKPNVDFYSASVYTSLKIP
RDLFTPIFAISRTSGWTAHILEQYDNNRLIRPRAEYTGPTNQKYIPIEER
>lcl|NZ_CP026520.1_prot_WP_042231152.1_1677 [gene=icd] [locus_tag=PC41400_RS08650] [protein=NADP-dependent isocitrate dehydrogenase] [protein_id=WP_042231152.1] [location=1902246..1903544] [gbkey=CDS]
MPKFENYALPTEGQAITIENGKLNVPNNPIIPFIEGDGTGPDIWAASQRVLDAAVEKAYKGEKKIAWYEVFAGEKAFNTY
NSWLPNDTLEAIREYIVAIKGPLTTPVGGGIRSLNVALRQELDLYTCLRPVRYFNGVPSPVKHPELVDMVIFRENTEDIY
AGIEYQEGSPEVKKVIEFLQNEMGANKIRFPETSGIGIKPVSKEGSERLIRAAIEYAIKHGRKSVTLVHKGNIMKFTEGA
FKNWGYALAEKEFGDKVFTWTEYDRIKEEKGTDAANQAQKEAEEAGKIVVKDAIADIALQQVLTRPTDFEVIATLNLNGD
YLSDALAAQVGGIGIAPGANINYMTGHAIFEATHGTAPKYAGLDVVNPGSVILSGVMMLEHLGWQEAADLIYKGIESAID
AKTVTYDFARLMDGATKIKCSEFGDEIIKHMG
>lcl|NZ_CP026520.1_prot_WP_042231151.1_1678 [gene=mdh] [locus_tag=PC41400_RS08655] [protein=malate dehydrogenase] [protein_id=WP_042231151.1] [location=1903701..1904645] [gbkey=CDS]
MAIRRNKITVVGAGFTGATTALMLAQKELGDVVLVDIPQLENPTKGKALDMMEASPVQGFDSNIVGTSSYEDAAGSDIVI
ITAGIARKPGMSRDDLVNTNAGIVRSVCENVKTHCPDSIVIILSNPVDAMTYVAYEALGFEKNRVIGQSGVLDTARYCTF
IAQELNVSVEDVRGFVLGGHGDDMVPLVRYSNVGGIPIEKLIPSERIEAIVQRTRTGGGEIVNLLGNGSAYYAPAASLVQ
MAEAILKDKKRIIPAIALLQGEYGYDNLFMGVPTLLGASGIEKVFELELTSEEKAALDKSADSVRNVIKVVTGA
>lcl|NZ_CP026520.1_prot_WP_241688823.1_1679 [locus_tag=PC41400_RS08660] [protein=hypothetical protein] [protein_id=WP_241688823.1] [location=1904756..1905169] [gbkey=CDS]
MLFLQTLLFLHVTGAITLGFYLLLPLLLHSSLKQPEDARLGQLRLIRHLNRVGQWILLAQLATGGYLISLGTYSAAWITA
VITLFTVVGAAGGMLGVALRRVINGRTEAMRQAFWYSAAAGLAFVLLLVLKKFPQIL
>lcl|NZ_CP026520.1_prot_WP_042231149.1_1680 [gene=yfbR] [locus_tag=PC41400_RS08665] [protein=5'-deoxynucleotidase] [protein_id=WP_042231149.1] [location=complement(1905345..1905926)] [gbkey=CDS]
MEHHFFAYLYRMRYIERWSLMRSTVRENVAEHSFHVSLLTHVLCTIGNEVFGRSWPTDRVVSMALFHDATEVITGDIPTP
VKHHNPHILRNFRELEQLAADKLVQMIPDELRHAYEPLILGPADPELARIVKAADLLDAYLKCAAELSAGNREFGTAKKQ
IEQSITELNMPEVDYFLTRLAPSLEKTLDELSD
>lcl|NZ_CP026520.1_prot_WP_042231148.1_1681 [locus_tag=PC41400_RS08670] [protein=polysaccharide deacetylase family protein] [protein_id=WP_042231148.1] [location=complement(1905928..1907109)] [gbkey=CDS]
MYKRLLSALLLAVFGLSALLIPTGADRYQMWYKDQVAVLMYHHIHDQDTSSSTITSKLFRDQMNYLLAKGYHFITLQEFK
QYLAGASVPDNAVLVTFDDGYESVYSHAYPVLKELKIPAVNFIITETLADPKGTMIPSMSKEELTRMSHETNFIENQCHT
NASHRKLDSGEAALVGRETVNGVQETDEQYRSRIYNDAAACASNLSPLQELPVDTVAYPYGIIADEGAEMYSKAGFRYGF
TIIPQMATRDADRMHIPRINAGSPWITPERLHTTLKRRAVAQSQTSDSVRLTEAVDHLGGAVTDDGQSVTIRSDNRVFSG
SIGSRRFKSASSVVELSEPVRREHGELVIGLSDLERLLGREVTYNRSTGQIYWRQEPRILTSQDSKSHQGGTP
>lcl|NZ_CP026520.1_prot_WP_042231147.1_1682 [locus_tag=PC41400_RS08675] [protein=fumarate hydratase] [protein_id=WP_042231147.1] [location=1907262..1908845] [gbkey=CDS]
MQHFLQSVYDLIVETSTNLPADVRRAIRNAQSLEDKGTRAALSLSTIAQNIEMAETNVSPICQDTGMPTFIIHCPVGADQ
ITMKRQIREAVAQATTDSKLRTNSVDSLTGANTGDNLGPGTPVIHFEQWERDDIEVKLILKGGGCENKNIQYSLPAEIPG
LGKAGRDLDGIRKCVMHSVYQAQGQGCSAGVIGVGIGGDRTTGYELAKHQLFRSLDDVNPNEDLRKLEDYVMENANKLGI
GTMGFGGQVTLLGCKVGVMNRLPASFFVSVAYNCWAYRRQGVVLNAQTGEIEDWVYERGSDQPMNEATQPVAAAEASQPE
AVAAAPDADESDERREIVLQTPISEEQIRQLKVGDVVIINGMMHTGRDALHKYLMDHDSPVDLNGAAIYHCGPVMSKDQA
GNWKVRAAGPTTSIREEPYQGDIIKKFGIRAVIGKGGMGPKTLAALKEHGGVYLNAIGGAAQYYARCFKKVEGVDFMEFG
IPEAMWHLQTEGFAAIVTMDSHGNSLHADVEKNSLQKLEQFKEPVFA
>lcl|NZ_CP026520.1_prot_WP_042231146.1_1683 [locus_tag=PC41400_RS08680] [protein=ATP-binding protein] [protein_id=WP_042231146.1] [location=1909225..1911024] [gbkey=CDS]
MLKFRARLTLILMLMIGVSMLIAGIFMAKLLEDSHIKELEANLVRELRVIEMTGDWNKAGDLAFLVPYYTSQTKRLKEFT
DARVTYIREDGKVLGDSDENPEELDNHAGRSEIQEANREGVGYAIRYSDTLKRNMLYAAIPVKTAGGEKTGYLRLAISLE
QVENTIRHLWLFLLIGLGILFLIAGVTSYRIARGLTRPIENITNVAKQITNRNYKARVETKSLDEVGQLGQAINRMADSL
HMQMNRILENEGRLKGVLENMGSGILMVDRDQRIVLVNRSAEDILGFSSNELYGKKYDEAKQQFELTRLIQECIETNEPI
REEIIFYYPSERILEINLSPISQADGEWSGVLVVLHDITAMRRLERMRSEFVANVSHELKTPIAAVKGFSETLLAGALDD
KETARSFLQIIFDESERLNRLVGDILELSKIESKRIPLQLSPTHLQSIIQKSLDMMKAEADKKKIALSMNVDEALFLEAD
EDRLRQILINLMANGINYTPEGGRVSVSAELAGSGTDGEDERIRISIKDTGIGIPKKDLPRIFERFYRVDKARSRSSGGT
GLGLSIVKHLVEMHHGTIRVESEVGYGTEFIIELPVIHP
>lcl|NZ_CP026520.1_prot_WP_009676698.1_1684 [locus_tag=PC41400_RS08685] [protein=response regulator transcription factor] [protein_id=WP_009676698.1] [location=1911152..1911871] [gbkey=CDS]
MSQKILVIEDEPTLARLLSYNLTQEGYETKVVDHGGDGLQAAMQQPYDLIILDIMLPGLNGFEVLSKLRQKGNATPVVIL
TARNAEEEVVQGLKHGADDYITKPFGVAELLARVAAVLRRTQADEPGGEGKKSTDKVIAVGDLQIFPEKYEVYLGEESIP
LRPKEFEVLLYLVQRPGVVITRDDLMNIVWGFDYIGGQRTVDVHVSSLRKKLEMNQQSVQIESIRGVGYKLVAHQGRKV
>lcl|NZ_CP026520.1_prot_WP_042231145.1_1685 [locus_tag=PC41400_RS08690] [protein=AraC family transcriptional regulator] [protein_id=WP_042231145.1] [location=complement(1912465..1913334)] [gbkey=CDS]
MEIYNEQINYENPFLSMRVFQARRDDERTSPWHYHKEVEMLAIVEGELDVYLENERVNLTAGDVVLIGSSELHRDRSSAD
TTLVYYVFQFDLEHYFDESGIAYLRYFSETQSALSKLNYIFRENPAIRDEVFACIRNIYTEYTGKKVGYEIAISILVKQI
VLSLLRSDEKRLLDYGNSADLIRLKPVFEYIDRNIDGKVYVDEASKAANISYYYFVKYFKKVLGISFLDYVNFKKIKKAE
KILLTQDISVAQVGEQIGMPNMAHFYKTFKKINHCSPHEFRKKMLEWNR
>lcl|NZ_CP026520.1_prot_WP_042231144.1_1686 [locus_tag=PC41400_RS08695] [protein=Gfo/Idh/MocA family oxidoreductase] [protein_id=WP_042231144.1] [location=1913540..1914631] [gbkey=CDS]
MSQKYRIGIIGCGGIANGKHLPSLAKLPNVELVAFCDIVIERAEKAKEQYGSGDACVYQDYRELLKDSSLDIVHVCTPND
SHADITIDALEAGKHVMCEKPMAKTSVDARRMVDAAKRTGKKLTIGYNNRFRSDSQYLKKLCEDGELGEVYFAKAHAIRR
RAVPTWGVFLDEEKQGGGPLIDIGTHALDLTLWMMDNYKPKVVLGTSYHKLSHRANAANAWGPWDPAKFTVEDSAFGMIT
MENGATIILESSWALNTLEVDEAKCTLSGTEGGADMKDGLRINGEKNSRLYTQNIELSSGGVAFYEGASENAPDLEMRMW
IDAIENDKDPVVTPEQALVVSEILEAIYESAKTGKAVYLNEEK
>lcl|NZ_CP026520.1_prot_WP_255246846.1_1687 [locus_tag=PC41400_RS30560] [protein=hypothetical protein] [protein_id=WP_255246846.1] [location=1914780..1914908] [gbkey=CDS]
MGLLEGYLSFLLVIGCVVFKRFAIFFVVHRVGTSFRESILTY
>lcl|NZ_CP026520.1_prot_WP_255246845.1_1688 [locus_tag=PC41400_RS30565] [protein=hypothetical protein] [protein_id=WP_255246845.1] [location=complement(1915048..1915182)] [gbkey=CDS]
MEHFMIMYAPPIVVAVSLLFLFLWAPRAKDVTDEPESDRQRRMR
>lcl|NZ_CP026520.1_prot_WP_042231143.1_1689 [locus_tag=PC41400_RS08700] [protein=cytochrome d ubiquinol oxidase subunit II] [protein_id=WP_042231143.1] [location=complement(1915190..1916215)] [gbkey=CDS]
MNLELLGITVLWVFLFGYLIVASIDFGAGFFSYYSVVTGSKNITHRIIERYLSPVWEVTNVFLVFFYIGLVGFFPETARY
YGTALLVPGSLALILLALRGSYYAFSTYGSKDNQWYMLIYGATGLLIPAALSTVLAISEGGLIVETAGQITLDWSALLGS
VYTWSVVLLAIVSVLYISAMFLAYYAHKAKDEAAFEIVRGYALVWSVPTILASVLVFFAISRHNPEHFTKMLDLAWMFVV
SFLCFLIAVYAVWKKRHLGLGFTFVLLQFAFAFFGYGASHLPYILYPYITIHENFTNPAMGTALIIAFIAGLLILIPSLV
LLMRLFLFDAGYVQGNRDKKG
>lcl|NZ_CP026520.1_prot_WP_042231142.1_1690 [locus_tag=PC41400_RS08705] [protein=cytochrome ubiquinol oxidase subunit I] [protein_id=WP_042231142.1] [location=complement(1916212..1917567)] [gbkey=CDS]
MFHYDPVLYSRILTEVTLAFHIIFATIGVGVPVMIALAEWTGIRKNDRYYTLLARRWARGFVITVAIGVVTGTAIGLQLS
LLWPSFMEIAGQAIALPLFMETFAFFVEAIFLGIYLYTWDRFKPKRHLMLLIPVAIGSSASAFFITTVNAFMNTPQGFTL
ENGKITNVDPIAAMFNPATPTKVSHVLASAYMTCAFILAAIAAFSLLRRKGDRLYAKKALKLTLTAGLVFALATAAIGDL
SGKFLAAYQPEKLAAAEWHFETQTKAPLVFGGILTEDKEIKYGLKIPYALSILAGGTPGFEVKGLDQIPDELEPPLYVHY
FFDGMVTVGMFTIVIAALYLWLSRSKRASKPAPRWLLRVIVACGPLSMLGIWLGWFYAEVGRQPWILRGYMKVAEAATTS
AHVDTMLLLFCLLYLVLGLTAIKVLSKLFKNNDVADELIAFGVFEKRGPLE
>lcl|NZ_CP026520.1_prot_WP_042231141.1_1691 [locus_tag=PC41400_RS08710] [protein=hypothetical protein] [protein_id=WP_042231141.1] [location=complement(1918165..1919004)] [gbkey=CDS]
MVYAFTIQPGIYHAFEDEAEQIEQCKEWGLLSGKAIKTKAFFYKGNGLNVPCTIVGYTDKMTAVIELDNKQQHCIHPSYL
KEMQAANYGQRQWSSGEAAPEADASEENEPAGNVETAADEAVETSAGAVQQAQPTAEPVPAPQTESAAGQAAPADPDRPA
AASEAVDAPKAKAKKEKAPKLQLPEGKVKMTAKVQDFTTVPNHFSDNDDEVIIYENVAIVEPETDIGTAWSSHSATLKKL
ELEVGDTITFEAKIVAKKLTRYPVPYKINNPAKIQKEQE
>lcl|NZ_CP026520.1_prot_WP_042231140.1_1692 [locus_tag=PC41400_RS08715] [protein=HAMP domain-containing sensor histidine kinase] [protein_id=WP_042231140.1] [location=complement(1919033..1920568)] [gbkey=CDS]
MKQGRTLTRAARLLAPRSLQFQLLSRSLYILAALLVLVGALQSIWMKNFLYRNRAETMSIQLNGLPRDLLDRSSLLPERH
GKSKEEEEVQPGMERPRGPVLFLPDTSLAYIAPDGTFTDLTKESGMISPQLPAVEYTAVREKMAPRRPASYKIVQDAAGT
EQLVVFRPAERSGSFGGIIQMGTGTAQLQAVVLQQLWIYGALVVLALAGGLALNIPVMRRTLQPLNKMVEAVEQTDAGNL
NERFPETQGQLEIDSLAKSFNGMLQRLETSFIAERDAKEQMRRFIADASHELRTPLTSIHGFLEVLLRGAADNPKQLYAA
LTSMHGESTRIKKLVEDLLTLAKLDRAPQLQLSDTRLDRLLLEMEPHLRMLAGTRTVHIDLTAGVRGTFDPDKMKQVILN
LFHNAVQHTDPETGVITVILSADRDGISLKVKDNGPGIPEEHAAHVFERFYRSDSSRTRKHGGAGLGLSITKSIVEAHGG
TISVRSREGAGSTFVLTFPENGENKGGKVLK
>lcl|NZ_CP026520.1_prot_WP_042231139.1_1693 [locus_tag=PC41400_RS08720] [protein=response regulator transcription factor] [protein_id=WP_042231139.1] [location=complement(1920565..1921251)] [gbkey=CDS]
MKNVKDIRILLVDDEPHILQFLEFGLVNEGFEIRTALDGMSAINLVREFQPHVVILDVMMPGMDGFEVCRLLKKNGSNIA
VIMLTAKDEVEDRVKGLTIGADDYLVKPFSFEELLARIEARLRNQFPGVFREIAIGPFRIDDPRKEIIFGERSLELSPTE
YELLKFLVVNHGIVLSKTKILEKVWGYDFGGEENIVEVYIRSLRDKLGDKEHRLIRTLRGAGYRLDWS
>lcl|NZ_CP026520.1_prot_WP_042231138.1_1694 [locus_tag=PC41400_RS08725] [protein=glycosyltransferase family 39 protein] [protein_id=WP_042231138.1] [location=1921598..1923979] [gbkey=CDS]
MISNRRFRVDLPLVFILLLGVFLNTYNIWLDAYANTYYTTAVASMLQNFHNFFFGSLDSAGSVTVDKPPVTFWIQTVSAY
IFGLHGWSVILPQALAGVGSILLIYLLVKPSFGTAAARISALVMACTPVAVAVSRTNNIDSMLVFTLLLGTWFLFRAIKR
NSVLSLLGAFALIGLGFNMKMLQAYMILPAFVLFYLFAFKSTWKKKAGVLSGAIALMLVISVSWAVIVDSIPASERPYIG
SSETNSVLELAFGYNGVSRLTGNQGGGPGGERGGMPGGERRGVDGRTGDAVGTAPGTAGTDSGSAGTTGGQSAAGGAGDN
GTLPGVGGATGGSAAVPDGANGGGMPGVPGGDGQGQPPGFGGGPGGGSGGMFGTGEKGPLRLFQSALSGQASWMIPFAAL
ASVALLAGLRRNNFTRKHKESLFWLSWLLPGMAFFSVAGFFHHYYLIMLVPPIAALAGAGWSEMWSAYRSKSGWLSWLLP
AAVAVTAAFQVFIMAPYNGTIGSGWTYGVGLTGAAAAIVLIVLKFRITRFSYAAAVAGLLVLLIGPAYWAATPITYGLNS
MIPQAGPSGGSDRGAMGQWPGGGMMPGGGSDGAGSGPQNGDGANQGAGPGTQEAGSGDGPDASTGGASIGSETDRSAGGW
GAPGAMGGGMGGRSVNEALLAYMKAHIGSTTYLFATSDYNTAAPYIIDKGEKVVTLGGFSGNDPVYTTAQLEDMVKTGKL
KYFLISEGGGGRGGNAELTQWLAQHGKKISSEEWQSGSAQTQTDTGAAAGMDGGRMGSMTLYEVTIEDGGNAS
>lcl|NZ_CP026520.1_prot_WP_042231137.1_1695 [locus_tag=PC41400_RS08730] [protein=glycosyltransferase family 2 protein] [protein_id=WP_042231137.1] [location=1923976..1924956] [gbkey=CDS]
MTPKVNYSIIIPMYNEEAVIQETYRRLKVVMSEASETYELIFINDGSRDRSAEIIKEYAAWDETVKLIDFSRNFGHQIAI
TAGMDYASGDAVIIIDADLQDPPELILQMIGKWKEGYEVVYAKRVKRSGETFFKKWSASLFYRVLRASTDIHIPVDTGDF
RLMDRKVCDEMKKLPENNRFVRGLVSWVGFRQTAIEYERDERLAGETKYPLKRMIKLSLDGITSFSYKPLKLAGYLGVLL
SGSGFLYLLYVVYLALFTDATLKGWPSVVGIMLIFNGFVLLMLGILGEYIGRIYDETKGRPLYIVRECHGLQKQEGPMPK
RLARYE
>lcl|NZ_CP026520.1_prot_WP_241688824.1_1696 [locus_tag=PC41400_RS08735] [protein=GtrA family protein] [protein_id=WP_241688824.1] [location=1924949..1925428] [gbkey=CDS]
MSNSRDLSNQGGPGGLSSPDRLGSRSNRLLPLVKFGLVGLLNTGVDFAAFLLLTEAGVSYLTAQVIAFLLGVLNSYFLNR
SWTFRSRTGRGTREIARFFLINLISLAVSTVLLQWAHSGLGWPLVTGKLAATFASSWINYAGSRYWVFRLPTPTGSEKS
>lcl|NZ_CP026520.1_prot_WP_042231136.1_1697 [gene=galU] [locus_tag=PC41400_RS08740] [protein=UTP--glucose-1-phosphate uridylyltransferase GalU] [protein_id=WP_042231136.1] [location=1925425..1926336] [gbkey=CDS]
MTIKKAVIPAAGLGTRFLPATKAQPKEMLPIVDKPAIQYIVEEAVRSGIESIIIVTGRNKKSIEDHFDRSVELEQTLSEK
GKMKLLQEVQAISEMAKIHFIRQKEPLGLGHAILCAKQFIGSEPFAVLLGDDIMVSDPPALRQMIHLYEETGRQVIGVKP
VPDSEVDKYGIIDSAGGRNRIHRVAGLVEKPSLGSAPSNVAVMGRYVLEPTIFPILENIEKGASGEYQLTDALHEIAKRE
ELLALELAGNRYDIGDKLGYLKAVLEIGLKREELASQLLPYLKNLVYQQEAEAWKLQRERRRA
>lcl|NZ_CP026520.1_prot_WP_241688825.1_1698 [locus_tag=PC41400_RS08745] [protein=DUF4405 domain-containing protein] [protein_id=WP_241688825.1] [location=1926430..1927113] [gbkey=CDS]
MKKITFIKLALDIVMALTLVLLFNKRVLGGLTFHEIAGLAISFAFLTHILLNLQWVKKVTLRIMDRNLPGKTRFGYLLNV
LLLVTMTFIIVSGILISEVVFRGIHLGEERWFKMAHISISYFVLVLIGVHVGMHWQWVIQVTGRIFPERFRGKAGAAAAK
IAAAAIVAFGVYEMYTTNYLTRLEGVGSVLGITSASSPMMQEGGGRRRPAAAAEHVHRWGQSAARGF
>lcl|NZ_CP026520.1_prot_WP_241688826.1_1699 [locus_tag=PC41400_RS30220] [protein=hypothetical protein] [protein_id=WP_241688826.1] [location=1927193..1927423] [gbkey=CDS]
MQERGPGAERAAVPGGFHPGEGRGGGGFASPNPLGVIAVYSAMMGVFVAITYYVEKRGRRRKHAKSGTAQSKPKVV
>lcl|NZ_CP026520.1_prot_WP_042231135.1_1700 [locus_tag=PC41400_RS08750] [protein=hypothetical protein] [protein_id=WP_042231135.1] [location=1927651..1928034] [gbkey=CDS]
MAIKHLVLELYCTQTWSDANGTVAPSFCKNGIGEPGYHCFDNCCTYMASTCAPHEIAYAGEYGEVLDMDAWIGFGGEMIP
ADADEGRISELKKTWEEVCRRKIQEAYEEYLKLMGDQPPEFGDKGQL
>lcl|NZ_CP026520.1_prot_WP_042231134.1_1701 [locus_tag=PC41400_RS08755] [protein=hypothetical protein] [protein_id=WP_042231134.1] [location=1928031..1928258] [gbkey=CDS]
MITVFLLQHSYEWNENENTKIIGIYSSKEKVEAVIEKYKELPGFKEFPNDFYIDAYRVDENEWTEGFMFGLIYGN
>lcl|NZ_CP026520.1_prot_WP_042231133.1_1702 [locus_tag=PC41400_RS08760] [protein=tail fiber protein] [protein_id=WP_042231133.1] [location=complement(1928401..1928943)] [gbkey=CDS]
MADAYLGEIRIFAGNFAPRGWALCNGQLMPVSQNSALFAILGVQYGGDGKTTFALPNLMGKAPMHQGAGQGLTPRVVGQS
VGSPAVTLLTTEIPSHTHIPQSVNAPGSDTNPTGNFWAQSPPEGRPGRETQRNLYDPTPNVPMNPLNLSVTGGSQPHNNM
QPYLAMNFIICLSGEFPPKQ
>lcl|NZ_CP026520.1_prot_WP_042231132.1_1703 [locus_tag=PC41400_RS08765] [protein=tail fiber protein] [protein_id=WP_042231132.1] [location=complement(1928962..1929462)] [gbkey=CDS]
MSEPFVGEIRLFANNYAPRGWAFCEGQTLSINGNQALYSLIGTVYGGDGKTNFKLPDLRGRVPIHVGTTNPLGQSEGEAA
HTLTINEMPAHSHQVSASGNPGSTFAPAANVWADETALYEPPSTPAVSMNAGALSIAGQSQPHSNMQPYLVANFAIALVG
IYPSRN
>lcl|NZ_CP026520.1_prot_WP_042231130.1_1704 [locus_tag=PC41400_RS08770] [protein=tail fiber protein] [protein_id=WP_042231130.1] [location=complement(1929494..1929994)] [gbkey=CDS]
MSESYLGEIRMFGGNYPPQGWALCNGQLLSIAENEALFVLLGTTYGGDGRTTFGLPDLMGRIPVHPNQTYIRGSKGGTET
VTLNQTQLPQHTHIPYATANLGTENSPANNTWATAITQNYASSGTTTPMNPQTLYSVGGNQPHDNMMPSAVITFIISLYG
VFPPQP
>lcl|NZ_CP026520.1_prot_WP_042231128.1_1705 [locus_tag=PC41400_RS08775] [protein=GNAT family N-acetyltransferase] [protein_id=WP_042231128.1] [location=complement(1929991..1930476)] [gbkey=CDS]
MLKKISEPEDQPFLFELYCSSRMAEVRAWGWDEATLRTFLHMQYSAQSASYRQQYPEAESYIVCQDDQQAGRILIHRTPI
CIRIVDITILPQFQNQGLGTKLLQTLMEEAREQNLPLRLSVLRTGRALALYERLGFQNAGENDMYLAMEWNHETCKEGDR
S
>lcl|NZ_CP026520.1_prot_WP_053228740.1_1706 [locus_tag=PC41400_RS08780] [protein=cadherin-like beta sandwich domain-containing protein] [protein_id=WP_053228740.1] [location=1930751..1936837] [gbkey=CDS]
MKELGKTGLWGRFLRIALVFLLVAGTGGAGNWIGTAEAAGTWDTVPNNAGTDLNQVAYGNGKWVAVGNMGTIMTSSDGLS
WSKTTIGLPSAVGVNYAGNQWIVTGAGGKIATSPDAITWTVRTTGASASLSAAAYNGTVWVIAGYSGVILSSPDGIVWTP
RTSGTAANFYSVTYGGGLFVAVGGSGTIRTSPDGITWTTRSGSPEDLMGVNYGAGQFVAGGSNQRVLTSSDGITWTPRTV
SVAGMNLYGIAYGANKFVAVGDTGGGPGTIASSPDGITWSPEKTGGDFLNHVGFANGRFIAVGGSGAMLTQFNTFSTNAN
LSNLTLSTGTLSPVFASGTTSGYKADVSNGTASINVTPTVADSTATLKVNGSAVASGSPSPVSLNVGANTITIEVTAQDG
TTKKTYTVEVTRAASTNANLNNLTLSAGTLSPTFASGTTSYTAGVANNVTSVDVTSTLADGTATVKVNGSTVTSGSPSPV
SLNVGANTITVLVTAQDGTTQKTYTVTVTRAAPLSTNANLNNLTLSAGTLNPSFASGTTSYTAGVANNVTSVDVTPTLAD
GTATLKVNGSTVTSGNASPVSLNVGANTITVLVTAQDGTTQKTYTITVTRAAPLSTNANLNNLTLSTGTLNPAFASGTTS
YTAGVANNVSSINVTPTVADGTATVKVNGSTVTSGSPSSVPLNVGANLINVVVTAQDGTTQKTYTLTVTRAASNNADLNG
LTLSAGTLNPSFASGTTSYTASVVDSVTSVDVTPTLADGTATVKVNGSTVTSGSPSSVPLNVGANTITVLVTAQDGTTQK
SYTVTVTRAVPLSGNADLSNLVLTGTALNPGFAPGTITYTSSVAHGVTALDVTPTVADSHAAVTVNGQAVTSGSASSVPL
NVGPNTVTVLVTAENGATKTYTVTVTRAAGSNNADLSGITLSGGAALSPVFAPGTTLYTVNVPNSVFGLDVTPSVADGTA
TLKVNGTAAASGTPVPVSLNVGNNTVTLLVTAQDGTPKTYTLTITRAPSSNALLSSLTLSAGTLTPVFDSGTTSYTAQVA
TGVSSIDVTPVVADGTATLTMNGHAATSGTAVPVSLNVGDNTITVVVTAQDGTTSETYTVTVNRAAPLSGNADLSSLSLS
AGTLNPAFAPGTTAYTVNVANSVNTVDVTPATADSTAFVKVDGKSVTSGTASAVSLNVGSNTVEVQVTAQNGSVKTYTLT
ITRAASGNADLSGISLSEGTLTPLFGPAQLSYTAAVPNASTGIDVTAVTADATASITVNGDAVVSGTPKPVALNVGDNPI
TIVVKAANNTLKTYTVTVTRAASSNADLSALTLSKGTLTPAFSAGTKLYTATVASDVYDLDLTAQLADLTATMTIGGNPA
SSGQIVNISLIMGANPVTIVVTAQDGSTKTYTVTINREAPPSSNALLSGLTISPGTLSPVFAGTQTSYTADVAHDITEVT
VTPAAADSTATVSVNGHVLSAGGSEIVPLLTGANTITVQVTAQDGTKNSYVIIVTRAVPPSGNADLSGLAVSAGTLTPAF
APGVTAYAASVNHSVSEVTVTPSPADPTATVTVNGTAVPAGSGRQIALSVGANPVTIVVTAQNGTTKTYTVTFTRESSGG
GGGGGSTERPIDGRIEVVSSSGEVLNQSITKLIGDNVKLVADIYSTDGATLFKQNVSVDASGKFTLYSLSAGTYKIVLSL
TAPNGEKLAGAAGTLVVSSDTAVFRTPEINPFGTIRDSETGAALAGVKVTLYWADTEKNRSAGKIAGTPVSLPELAKMMP
GQNKNAQTSSSDGRYGWLVYPDGAYYLIAEKDGYETYDSRSDRGSGIVAKDFTSLKLDIALKPSGTYHEHKPYMEGYPDG
TFQPGKGISRAELAAILARLFANETSAAAQNPFADVPATHWASQYVTAVFGGKLMDGYPDGKFNPERILTRAEMATILVK
LKQLPAVQGTSAFTDVQGHWAAENIRKAEKAGLLAGFEDGTFRPDQALTRAQAVVIFNKLLGRQPESVPASIPQAWSDVP
ASFWGYKDVTEASVYHAYKHVNGREIWK
>lcl|NZ_CP026520.1_prot_WP_042231126.1_1707 [locus_tag=PC41400_RS08785] [protein=response regulator transcription factor] [protein_id=WP_042231126.1] [location=1937082..1937738] [gbkey=CDS]
MQKKILIVEDDIHISNIIKMNLNLVNYATAEVYDGISALDAVKQDKFDLILLDVMIPKLDGFALMEKMKSCGIPVIFLTA
RNLVYDKVNGLRLGADDYMVKPFEAIELLARIETVLRRYGKEERVIAFQNLQADLNKREVTLQGEPVELTPKEYDLLIVL
LRNKNIALSREQFIDKIWGSDYYGETRTVDMHIKSLRKKLHLQDHIKTIYKIGYRLED
>lcl|NZ_CP026520.1_prot_WP_042231124.1_1708 [locus_tag=PC41400_RS08790] [protein=HAMP domain-containing sensor histidine kinase] [protein_id=WP_042231124.1] [location=1937742..1939181] [gbkey=CDS]
MKFWQKIYLFSILVFVFIFNAASVMVIERSHGKMLDQEINGALSQNMSIHSSVDAIVPILRIYDSIDYEKTVLTNMANEF
VAKNSDQRIYLEILDDTNRVIFSNNDFKMPEHREELDKLAMDEIHYILRDIGTRTLLFTSNHTEINHKSYVFTYTKDVTP
LYQDRMDQYGFFIKVDIAACLLYMLIMFFVSRGLTKPIDRLNRTARVIAQGGFSERVRLKSKDEIGVLAQNFNEMASAVE
EKINDLKRHNEEKQRFIDNFTHELKTPLTSIIGYANFLRTTKYNEALYVDGLNVIYSEGKRLESLSMKLMDLIFLQEDHF
RLEEHDLGTVIAEIKPALAMMAKEHRITIVTDLEEGRLWLEKDLIKMLIFNLVDNALKASAEQQSITIRTYWRGSRCTLE
VIDQGIGIAEEHRDKIFEPFYMADKARTRSSNGAGLGLSICQSIASIHHAVIEVSSLENQGTTIRVTFDRTNPKGGMNP
>lcl|NZ_CP026520.1_prot_WP_042231122.1_1709 [locus_tag=PC41400_RS08795] [protein=hypothetical protein] [protein_id=WP_042231122.1] [location=1939178..1939930] [gbkey=CDS]
MNKFIAVVLCASFLVLTTGCEGLFNTVKKDTVLVKKISQEHKMSEVKYEGVLAQDTVKTLSLNAVNKTYNEKLKLEEVRF
ELMAVDRKKLEDLLSKTEYGVRPMLERNQKVKIDFETELNKVPGGLFYVTLTRVADPDEVYELVLNARDGEVLKVSSDIS
RPAPIEKEAIVIPDGRIETANRFIQEKGAYALSDLIPEEKIVRQGVVGEVYYKNKETESLAYSVRVNLRTNQVIGFSKDI
MAILGYFSGS
>lcl|NZ_CP026520.1_prot_WP_241688827.1_1710 [locus_tag=PC41400_RS08800] [protein=N-acetylmuramoyl-L-alanine amidase] [protein_id=WP_241688827.1] [location=1940168..1940836] [gbkey=CDS]
MLTGMVWSLYKLSASLTLASASIPSTELAVSPGKSHAALQNGTSRYKIVIDPGHGGKDPGAAGVSGKQEKAYTLALSKKV
FDQLRQDPAFDVVMTRTDDTFVELGGRAQIANELGADAFISIHGNTYKDPDVSGTETFYYADDSSALAHKVHEQLVKTTG
FKDRGVKKEGWQVLRDSKHPAVLLEVGYLTNPDNETDMLNEARQIRTAEAIANGIKKYFTEF
>lcl|NZ_CP026520.1_prot_WP_042231117.1_1711 [locus_tag=PC41400_RS08805] [protein=extracellular solute-binding protein] [protein_id=WP_042231117.1] [location=1940867..1942267] [gbkey=CDS]
MRKISFILIAVILMISASACSSRTGNGEAAGTQNGDKTTESPDGGKIEEKKSDGGPKTIVFSTFFPDDFFKDAKKRYEAK
HPNITIELKYVETDDAHGEENSEKFIKTTNTALLSGKGPDLIEMDLLPIGNYVNKKLLANMSEMMDKDPSFKREQYFTNI
LDNVKLNGGVYGMPLNFFTYAFIGNETAIEKSGLKIEDKNWNWSQFADVAKELMKDGNPKNKSALFSTPEYLLNERVKEQ
YAAFVDPVNRKANFETAAFTGLMKQVKSLFDDQIATEKPVPALFRSESIVSPEYYIRDLKQSEYLSKGYEFTSKLYAKPH
AEGQKAGGYFRTYKTIGINEKSSVKEEAWDFLKFMLSDEVQQGPDHAGFPLNKSVYEKRVKELLKVGKVKSDQEMGVLKD
KVFDITAKDIQDLDRFLSEAIYPVEFKPSKIEEIIMEEAKSYFKGQKSAEAAAKLIQNRVTTFLNE
>lcl|NZ_CP026520.1_prot_WP_042231115.1_1712 [locus_tag=PC41400_RS08810] [protein=sugar ABC transporter permease] [protein_id=WP_042231115.1] [location=1942288..1943181] [gbkey=CDS]
MADKLYRSNPFKRRGKNWIDVFLFLGPSLAGFAIFYLIPFAMGFFYSFQDSTLDGSFVGLENYKELLASGSFRNAAANTF
LFTFVSVPLILVLSLIAALLLNRKLFIRNWLQTAFVLPLVVPAASVVMIWQILFDWNGSVNAWMQSLHLERIDWMKSDWS
VGVLAVVYSWKNIGYNIILFLAGLQSIPKDYYETADIEGAGTFRKTIHITLVYLTPTMFLVVLMSIINSFKIFRETYLIA
GDYPYDRIYMLQHYMNNMFLSLDIQKLTAAASLMVGCILILVTLMLFIERRFRQFID
>lcl|NZ_CP026520.1_prot_WP_042231207.1_1713 [locus_tag=PC41400_RS08815] [protein=carbohydrate ABC transporter permease] [protein_id=WP_042231207.1] [location=1943238..1944065] [gbkey=CDS]
MAVIAAAMLFPIGVTFVNSLMTESEIAHNYELIGKMSDTAAEEGFVNLKLIPDWLSFQQYMEVLVYKPVFLQMFWNSAGL
VVPIIAGQALVASMAGYAFAKLRFWGRDKLFVVYLMTMLMPFQVTLVPNYMMAQKLGLLNSASAIILPGIFSAFGVFMLR
QFMLHIPNSYIEAAQMDGAGHLRIFYKIILPLIQPGIAGLIVLLFVDNWNMVEQPLIFLEDSFKQPLSLYLSRINEGARG
VAFAASVLYMTPMVLLFLHAESHFIEGIQLSGIKG
>lcl|NZ_CP026520.1_prot_WP_042231114.1_1714 [locus_tag=PC41400_RS08820] [protein=efflux RND transporter periplasmic adaptor subunit] [protein_id=WP_042231114.1] [location=1944116..1945285] [gbkey=CDS]
MELLHAGQASDRRKRKIRLFSGLFIGLLVVFTLLSNTFMALTIPKVAVVVPSRGQLVHKFKGSGVIKWRAEITLTNSTGW
KVKKVEVKEGDRVKKGQTLVTYDSHDAEQDILDQQAGLNKLKLTMEELQSHFIEASQNGEEKSIEDAKRAMDIHKIDLEV
QQRKIQNLQEKLKKNVKLTAPFNGIVTKVIAKEGLSSENGGYDVRLSDESLGFEFELSVPVDTAAELKTGDKLEVQVDGP
AAEKIEGQISEIREPGIPDPSKTEEGSTGEGGKTAMNLPTKRLLVTMQGTGTGLKEGAGVRVELSRATDPDILLVANKAI
HEDNTGKYVFTIEERNGPLGNAFYIHRTSITVTDSDDKVSMVTQGLFEQQQIVVESSEPLQDGDKVRMQ
>lcl|NZ_CP026520.1_prot_WP_042231112.1_1715 [gene=gap] [locus_tag=PC41400_RS08825] [protein=type I glyceraldehyde-3-phosphate dehydrogenase] [protein_id=WP_042231112.1] [location=1945530..1946597] [gbkey=CDS]
MSQAAGIRGIGISGMGRIGRLVVRKMFSDKGQTSPLKAINTIYPAETLAHLLKYDSVHGRWDADVTIADGCLVINGQSIR
LTYEREPENILWEQMGVDLVIDATGKFNDRQGSQRHLSAGASRVLITAPGKQMDFTVVMGVNDHLYDPIKHTLVSAASCT
TNCVAPVLHILDQAFGVQSGWMSTVHSYTSDQKHLDNPHNDLRRARACTESIVPTTTGVGKALTDVLPHLAAHVQGVSLR
VPTQDVSLVDLTVKVSRPAGLDEIQAAFKSAAADSMAPYVAYSDEPLVSVDYVGCPNSAVVDGLSLMTMGDQIKVLAWYD
NEWAYACRVVELAELMRDKAKTIQSRDAEVSSMFA
>lcl|NZ_CP026520.1_prot_WP_084706629.1_1716 [locus_tag=PC41400_RS08830] [protein=RNA polymerase sigma factor] [protein_id=WP_084706629.1] [location=1946716..1947261] [gbkey=CDS]
MDDEYLKYVSELHPSDLRTLMEQHGEDVWNYAYFLTKKRDLADDIAQDVFVKAFRSVGSFRGECSVKTWLLKITRNTSYS
YRRKAFFRRVILADYVRGESPSAESEFLERHLADEAWKIVLELPAPYREVLVLQSMHGMTVAEIAAALGISEGTVKSRIH
RARRKAAQLAKGGFRYEENGN
>lcl|NZ_CP026520.1_prot_WP_042231109.1_1717 [gene=lepB] [locus_tag=PC41400_RS08835] [protein=signal peptidase I] [protein_id=WP_042231109.1] [location=1947242..1948474] [gbkey=CDS]
MRKTEIETDPFELSERPFNKPMFGTESMDAVERKVWNGGGLEAGRPASRRKRLYLAGTALTALLVCCAVLFMGPLGDKAN
GTIRDLTTPLYPQRVEAKEGLMLVDDYSMSMVGQKMEYYSGGYGRLVVDPRFKMVQQFRRGDMLYYRLPAIPGFTPNPEI
SGSDSQTARVIGLPGEKIKIEKGRVYVNGKKLDTFYGQAKIVGLDEAAYFKNMKASGRKDACPPDCVKDTKNYFNANMAE
MTVPDGSLFVLGDFWSKSHDSREFGALTQGNIQGKVLGYDEAGARSPHQTFGSGNMWFTGIPGKLGILNGRISTSTPNKI
VWHFWGAREEFEGKNLRIEGVHTSDGVKKPVLLQEDGSVIWEYEWGPAYGPLNGADASLPSLIKLDESEQWRLDIYLGGK
FFASLFVNAE
>lcl|NZ_CP026520.1_prot_WP_042231107.1_1718 [locus_tag=PC41400_RS08840] [protein=YafY family protein] [protein_id=WP_042231107.1] [location=1948751..1949698] [gbkey=CDS]
MNKTDRMLAIILRLQRKGTQRAEDLAAAFETSVRTIYRDMQALCEAGVPVVGAPGQGYSLVEGYFLPPVSFTAEEAVTLL
LGLEFVEQQFDEASRAQAGAARAKLEGVLPDRVRRESQRHREGMKLLAGPGSAAGNGAAALALLRGAVLQERTVRFAYAA
KDTAMPGGERLTVREADPYGLVYLNSAWMLIAFCRLRQEVRHFRLSRISELTLLETVFEKPEEFDLRLYKPADDRQGEVR
LLFRRELANRVRETNYYYIEETQNVEQGLLVTLKVRSPEEILHWVLSWGAGVEVLEPESLRVRLREEIRRMAKRY
>lcl|NZ_CP026520.1_prot_WP_042231103.1_1719 [locus_tag=PC41400_RS08845] [protein=DinB family protein] [protein_id=WP_042231103.1] [location=1949764..1950342] [gbkey=CDS]
MQTNQNQSQRLNTIDILNRFEQLTETYIQQLENYSLEELTRKPDEDQWSLGQMYVHLIQSVLYRHLPIMEACRNSSGSEE
AAGKTESGVEAYARGSYPPIRIQVPPSKEYTPLQPESKEQLVSGLRDVIRAMRAIQPTLADIPTANTVTHPGFGELNAVE
WFALVEMHYRHHLRQKERLDAFLRGTKEASEA
>lcl|NZ_CP026520.1_prot_WP_042231100.1_1720 [locus_tag=PC41400_RS08850] [protein=DUF4272 domain-containing protein] [protein_id=WP_042231100.1] [location=complement(1950606..1951784)] [gbkey=CDS]
MNQCTLYISIKGYEKITEAIRTAFGDKQVQMSPDGNQITVVDKKLFSKTTVTFSLMTLDGNPGEFEPMIRGMYGFFSQIE
TPHEHIQAKVLKQISALNAAVGIVSSKEIDPNTLNRILEAASRVHAVLFLPTGQLLNPEGQVVLSAQGESEVEDFTVTVS
TDLIDAHIQSSPSAEARKERNMALLTEQGVPVIEHLPVIDGDEFAAVRTKEEIAARATALCMVAMYGADVAEDTDTKEAK
EFIEGVIELYGASSFFTEKERAFLESDSPDPTDAIQFAWMYECYWVLLWSLGYAEELGYPDSVCDVQAAVDALRLAGNFE
TFLEQAKVRSKADILEEADRIYRYDWACVDARIKGNPAPAGLNDGVVLERHRALNWLVRYMDAEWDDVSTDT
>lcl|NZ_CP026520.1_prot_WP_042231098.1_1721 [locus_tag=PC41400_RS08855] [protein=PLP-dependent aminotransferase family protein] [protein_id=WP_042231098.1] [location=1951895..1953343] [gbkey=CDS]
MIGYPKLDERDPQPLYMQLYLHIKTGIIRGEIAEHTRLPSIRKLAGFLGLSTTPVESAYHQLLSEGYIASRPKSGYYVEK
LPEPYGKLGSLHMGELPLVQEEEPDAARKRRFRYDFHLSRNDFTAFPYGTWKKLANEMLAPDNENLLFFGDPRGERRLRE
QIAAYLRQMRGVECSPGQVVVGADQFNLGVVLSQILKRYGSRLAVENPGYPLIPSIFGRAGYEILPVPLEADGIDVERLH
AAGARLVTVTPSHQFPRGMVMPVSKRLKLLQWARDTDGFILEDDYDGEFRYHGRPVPALQALVPGAPVIYLGGFGQIMAP
ALCVSYMVLPDKLVPLFDALRREMMFEQSASRLHQHTLALFMERGYLEKHVRRMRKLYRKKHDRLTAAVGRIFGSRAAVI
GRDAGFHIVLRVRDERPEAELLRLAEADGVHVSSAAFYWPGTVPPDGEKDFMVGFSGIAEEDIEPGMRKLKEAWFGDANL
SS
>lcl|NZ_CP026520.1_prot_WP_241688828.1_1722 [locus_tag=PC41400_RS08860] [protein=GNAT family protein] [protein_id=WP_241688828.1] [location=1953426..1954115] [gbkey=CDS]
MNESVNESVSEPVNRAVINPVNEHVKGPGSTEPDRQKSNCSNKAGAPVPVRFLEGEKVYLRPIDTGDTETYFQQLFQPQT
RKLTGTQKHFTRNQIERYIEGKAQDTSSILLLIALTETDEVIGDIALQDIDSYNRSAGIRISIDSSRNQGKGCGTEALNL
MLDYGFGIVNLHRIELNVFAYNERALHVYEKIGFKKEGVQKDALYYNHKYHDSILMAILEHEYRELHGL
>lcl|NZ_CP026520.1_prot_WP_042231094.1_1723 [locus_tag=PC41400_RS08865] [protein=SDR family oxidoreductase] [protein_id=WP_042231094.1] [location=complement(1954220..1955041)] [gbkey=CDS]
MHNPLKTAIVTGGASGIGRALCKQLAAKNIYVVIADIHAAQGEELAKEIAGKGGLARFFPLDVTDASQVRKVIQETAGEF
GRLDYLFNNAGISMYGEFQDMSLNDWNKIVAVNFWGVLYGTHAAYPIMQKQGFGHIVNTASAAGLGPTPMVSAYTATKHA
VVGFTTSLHYEAEMHGIKVTTLCPGHVDTAIYENGYAIKLDKERINASVKQRKMMSPDQFAALALKGIEKNKVLVCPVPL
RITTKVFFHLFPAMERKLMRLVCRVMREAKPAN
>lcl|NZ_CP026520.1_prot_WP_255246844.1_1724 [locus_tag=PC41400_RS30570] [protein=hypothetical protein] [protein_id=WP_255246844.1] [location=1955518..1955643] [gbkey=CDS]
MQLNGIDFSAEEIEEIEFLKELCEGMTVDGAEIVGFKVLSD
>lcl|NZ_CP026520.1_prot_WP_164977115.1_1725 [locus_tag=PC41400_RS08880] [protein=kelch repeat-containing protein] [protein_id=WP_164977115.1] [location=1956034..1958508] [gbkey=CDS]
MSWIRKQKNRTLRVLTSFVLLLNALTLGATAAYSQSMSTVGFEGAASLPDHSLTNPMNLTDKAVLPGTDVQSSVYGNVYL
NGFAKTLGASFTVTPAAPIPVFQISQYGQSALNDGRVMVTVGEYNANYNLIYNPQLNTWESAAPMPVKRNRHQQSTLKDG
RVLVTGGFLPLEDYSPSHKAYLYDPQTNTWSEAANMPQGRAGHSQSVLSDGRVLVAGGEAMRETLDTAYIYDPVSNTWSK
VANLPFDLAGAGQSTLLDGRVMVVGGSSLVDREWRPSSKKSLLYNPTTNSWSYAENIPILGDNAGRPQSTLLDGRVLNNS
KGSYIYTPSTNKWTKAEIIPTGFYPMEQSLLPDGRICVSGPDGRGGNAVYFISFNTPPKLSIDNTDQTVWKEDGRGVVTL
SGKVSDMNNDPVTISATIAGITKTTIVSNTSSSPSWTLQWDLNSDRLPGGTYTNIAVTASDSAEAVTSTYTGLITVNHAP
NAPTNIQPGSFSPSNPQMIATGASVLSWTFNDPDAGDIQTAFQVVILSADGKSQIYDSQWVVSSANTFTVPANLINRGSV
AGWYVRVKDSKGAVSNFSPLAYVRINTMPSAELTSYTDGQTLTDNMLTFTWKYKDADQQAQTEYQVVGTNDGWKNWAYNS
GEIKSGSTSHVAGPITSGSWNFAVRVKDGTEWSEWSFRNNLMLPNAYEPNDTFEQAFFVKMNESYASAITTATDVDFYKY
TAKANGVDRLTLNVPKGQNYNVHIYDSSKRLLAAGIQETDIPENVLYEVTEGQTYYFKIFGVNGGFSGSSYSFFLSSLTL
RNDTRYEYDNNGNLIKKSSTAQSQ
>lcl|NZ_CP026520.1_prot_WP_053228739.1_1726 [locus_tag=PC41400_RS08885] [protein=polymorphic toxin-type HINT domain-containing protein] [protein_id=WP_053228739.1] [location=1958541..1963640] [gbkey=CDS]
MKKRNKQIFALVMVLTILLTCAQGMLVPHAYAKSDEAKAKKDKIQKYLDKGFNSQEVEQALSKNIKEADLEEALTKSKPV
NKSKEASSTIINEVKEKPAPSFRMKAAAAGPEPVPDIVVVNTKPDEAPYSVNLEHETISTLSGSLSVREMDMVLRGRNGL
NFSLTRSYDSSSAQFNQMASYRSTGENGTIPYEEKMFPIGKGWSWDISSVERHEDTKYLHLAGGGVYKITSSGLEGYPWK
DLTITEESGNLENRYVLKSTNGTKQYFNYDGKLVKIEDDYKNAIYFRYSDDPVYKNVLTSISSDVGNAITISYSPTAVVL
TNGSQTVTYNKSLQSYIEVLTSVIDPIGRTTVYDYALKSAKSYATNISVVEAENPYSLLTGVMHPTGAKSVYKYEDNQVI
RNFNYGYNGFYRIKSREDQVTLQDGSIQTYNRKDITYDGDMGSSYDKDMEFAVTVDNKLNQTTFRTKKDFIEREKPSYYY
NTNITEITSVDGKTHRTSTDHVYDEKRHLPSPDKTTVTKSVDGAAPIIFTHSKVYDDYGNITSSTDPAGVTTTFGYNDNH
LLTSILEPRSSSQSKYTLYTWNKHTVILKQMFEGSAAGAKLSETAYENTDAYGNVTQIRNKNSAGETVMKLEYSSDYQGA
FPTKTTMAVTNADGNKSSIVNQYEYDAIRGLLTRQTDGNQGVTTYQYDALGRVIKAVQPDNSSVSIQYLDLQNQIRQKDE
TGVETVTTWNPLGWKTEEGIMQGTYKAKAKYGYDTYGRQTWIEDAVGNRTQIGYDAWNRQNQITYADQAVSRIATDDYNL
TQTTTDPEGNIFRQTADKLGRIVKKEQVAPTAKVLGTYQYNFAGNVVASTDANGGTTEFRYDPLGQLTGVINAKKEETKY
TYDQLGRMIRITYPDNNTTQKKYDELGRLIQTTDANGKVEKLYYDGNNNQIRATDRNGTELSYVYDARNFLVEKKAPDET
IRYAYDLAGRRTRMTDNTGTTQYTYQPSTGQLSKVVYPDGKTLAYDYNALGLRSSLNDPSGQNTYYQYDSRNRLSSVSSS
LADEEAAYSYYRNDLIKQTRQRNGITTAYTYNGLQLQTLTHKQADGTALNTYAYEYDANENETAKTENGQTYQFGYDPLN
RITSSSQYRESYAYDARGNRTSFQTNHEVNSPDADYVYDKKDRLSAVKMKDGSTVSYKYNGDGLLYERTDKDGTERYYYD
GDQVIAEGDGSGSLKARYVRGRELVARESGGSKAYYLHNGHGDVVELRSETGTRLNQYAYDIWGNPTVSEENISNPFRYS
GEYWDSATKLQYLRARWYDPSMGRFINEDTYEGQLDNPLSLNLYTYVMNNPLKYRDPSGHIQQATDNYEGINGIGYLGEV
VTNPVTDAMANGVLGGMKAAGEILDFLVISDITTVTDPNSSKFDKALAAASFIPIGKIVKGGKIFITLKNGERKIERAVA
YSEANWSKASKIPCNCFTAGTKVLTDEGEKNIEDIEVGDMVLSKNEETGEQAYKEVTHLYRNDKEIIYELTIGDQAIETT
DNHPFWVEGKAWVLAADLQVGDKLQQSNGNNLTIDNIKIVKHDEKVKVYNFTVADFHTYFVSDLGIWVHNINCTYTLSAD
NIQHLGKHTKSEFSNQVPYLNDKQLQSKLNGNTFFNPNWSMDDIKIATQNGINEAIDKGFTSGHYQYKYNNETITIFFKN
GKYDTAYGSHVLTPADFGR
>lcl|NZ_CP026520.1_prot_WP_042231085.1_1727 [locus_tag=PC41400_RS08890] [protein=hypothetical protein] [protein_id=WP_042231085.1] [location=1963650..1964186] [gbkey=CDS]
MFDINYRIIIDVNRWRNYDLEQIDKEGGIEGFFQLIFNTEEYGYYHSKELAPDEQGFDIISTWFDNLLEVCLILETSKYV
ALKDIETYNTWIEFIPKEDQISISLMKSESFTSEFIILQPLNNSTYSEWKYVLIPKEEFRNIVIQRTRNFIDELSRINKH
FPQSQRIVSLYELLSKIK
>lcl|NZ_CP026520.1_prot_PC41400_RS30225_1728 [locus_tag=PC41400_RS30225] [protein=polymorphic toxin-type HINT domain-containing protein] [pseudo=true] [location=1964883..1965605] [gbkey=CDS]
MCNCFSAGTKVITDEGEKNIEDIEVGDKVLSKNEETGEQDYKEVTALHRNEKDTTYKLSVGNQIIETTDNHPFWVEGKG*
VLAVDLKVGDKLVQSNGNHLKIDNIEIVHHYEKVKVYNFTVDDFHTYFVSDLGIWVHNINLFCGSASDLADLAKNSKKID
GLQGVKVSADLIHDSAIDFVGKGAKVQKIDGGMWYVSKDGLRRVRTGQKSKGAYEANFETFSNKGFNEKDKIGNFHVEIE
>lcl|NZ_CP026520.1_prot_WP_042231202.1_1729 [locus_tag=PC41400_RS08900] [protein=Imm8 family immunity protein] [protein_id=WP_042231202.1] [location=1965632..1966012] [gbkey=CDS]
MVVPVLKATHTFFEPETEDDFCIAVVADIGVLNSDGADQFYFSVISPKAIIKELEDVKVLNGRGYLIVQKFNLDIVRKEI
NKILEDCIRPTWDEVAKAINRHLNWEYDNIQYETLEEAMARLNKDN
>lcl|NZ_CP026520.1_prot_WP_129112462.1_1730 [locus_tag=PC41400_RS08905] [protein=hypothetical protein] [protein_id=WP_129112462.1] [location=1966562..1967206] [gbkey=CDS]
MKEISLTLFTYQVFNKQNIAKSVFDSLSENGFLFTKMGVTEPLKVRYSREEAISLWCSETEGCYDIELEKMVGKAGNFMA
KGPKSLFMGSQWWHCPNKKKLNYIYLTFSVDVLDSKKQEVLYLFKEFIVLFEALYGEITHSDALKRQHVTGTPDLRLPGV
FWCNYYSNEYLALMNCREQVESFEWWKKEDMEERGHGRRNDDLPKRRPWRVGRA
>lcl|NZ_CP026520.1_prot_WP_042231081.1_1731 [locus_tag=PC41400_RS08910] [protein=hypothetical protein] [protein_id=WP_042231081.1] [location=1967389..1967766] [gbkey=CDS]
MIINGVSVGPYELGWGLQELKEKISISGFVEEELDNHYTITTNTLKFWVEKEQGEITQITVFGEYEGKFLGGRVISTNRV
YFNLLFVTIVDLEGLPSRSILYPTQCRINRAVSGLHYWSQVKEWI
>lcl|NZ_CP026520.1_prot_WP_156972829.1_1732 [locus_tag=PC41400_RS29720] [protein=hypothetical protein] [protein_id=WP_156972829.1] [location=1967809..1967955] [gbkey=CDS]
MSLKEIKHTIQNNRLDDIYLVGFIDSEEDGLAEFVASMNYLFLEFGDN
>lcl|NZ_CP026520.1_prot_WP_042231080.1_1733 [locus_tag=PC41400_RS08915] [protein=hypothetical protein] [protein_id=WP_042231080.1] [location=1968017..1968298] [gbkey=CDS]
MVDLDDVVPAKSRISNVIFNNPLLDNKVSEIVLFNMKKNTEGLVCDALKLVLLNKQEIFFDPSFLGINVGGSEVEELWRD
NQKEYYESVSTII
>lcl|NZ_CP026520.1_prot_WP_129112463.1_1734 [locus_tag=PC41400_RS08920] [protein=hypothetical protein] [protein_id=WP_129112463.1] [location=1968476..1968658] [gbkey=CDS]
MAVLSGGRAKRPGNALRNIKGLDDVLNVPSKLKGNKPEELYKYLMDNGYNPNYPQVRLEQ
>lcl|NZ_CP026520.1_prot_WP_042231079.1_1735 [locus_tag=PC41400_RS08925] [protein=hypothetical protein] [protein_id=WP_042231079.1] [location=1968706..1968972] [gbkey=CDS]
MKLIEKIERIFRDLKFEKELINDTFVFVCNGKYRKVTFIKKLESFVIEYADSYDEAEKNLYEDGDLYPISLGENELINRM
RNELVDSL
>lcl|NZ_CP026520.1_prot_WP_042231078.1_1736 [locus_tag=PC41400_RS08930] [protein=hypothetical protein] [protein_id=WP_042231078.1] [location=1969213..1969539] [gbkey=CDS]
MNLYTYVMNNPLTHVDPSGHIQQATDNYKGVNGIGYLGEVVTNPVTDAMANGVLGGMKAAGEILDFLVVSDITTLTDPNS
SKFDSGEFYPNWQNSQGRQDLDYHEEWG
>lcl|NZ_CP026520.1_prot_WP_042231075.1_1737 [locus_tag=PC41400_RS08940] [protein=hypothetical protein] [protein_id=WP_042231075.1] [location=1970694..1970921] [gbkey=CDS]
MKTVFLLQHSYEINQIEYTKTIGIYSTRDKAQDVINKYKELPGFKEYPEDFYIDEYHVDKNEWQEGFLSSSAFED
>lcl|NZ_CP026520.1_prot_WP_042231073.1_1738 [locus_tag=PC41400_RS08945] [protein=hypothetical protein] [protein_id=WP_042231073.1] [location=1971241..1971957] [gbkey=CDS]
MGRNKKIALVLVSFVAFVLFSLVYYVSEGSPIQRYKFKKELDSYIQEKFSQNIMTENIKYSFKNDIYWILAHPENNKELS
FEISQSWNDRGLWDDFLTASWRFEVNQELESFVKSVYAQKGTARINLGNARSAVDKYSTIPYGGVPSYQDIKQELKDSII
YIHIDRAFKKESTEDEYDKVYKVIQFIKAKGYKFEQISFSYKGEQKDSPTGKTFEFNSLDNVNSQDDIVKFELLSREN
>lcl|NZ_CP026520.1_prot_WP_042231072.1_1739 [locus_tag=PC41400_RS08950] [protein=Imm8 family immunity protein] [protein_id=WP_042231072.1] [location=1972338..1972718] [gbkey=CDS]
MVIPQLKRILISSEPQSDDDFVVPAIADIGPRDVDGVDYFYFRIMTPKRMLSILNEDKIMDGRATFIVKEFNLALVEKEI
NKILEDCIRSTWDEVAKAINRYLNWEYDNIQYETLEEAMDRLNKIK
>lcl|NZ_CP026520.1_prot_WP_042231071.1_1740 [locus_tag=PC41400_RS08955] [protein=hypothetical protein] [protein_id=WP_042231071.1] [location=complement(1972895..1973260)] [gbkey=CDS]
MRKAFRVYGLAAVILTMTATTAYAANQESTAYAKSGGNDAKSSYITGQGQMGRMKFNSSGGGYGFYEKVCSGSTTYGPAS
YIPASGGSVDELVPMPSNCQFKLHITSGLNFPVAGTIQNYE
>lcl|NZ_CP026520.1_prot_WP_084706627.1_1741 [locus_tag=PC41400_RS08960] [protein=immunity 53 family protein] [protein_id=WP_084706627.1] [location=1973638..1973928] [gbkey=CDS]
MEILRWLQEWFTSHCNGDWEHESGIRITTIDNPGWHVSINLRETELASRQVDVVQIERTDDDWVHCKIEDGCFSGAGGPG
NLEEILRTFYHWATND
>lcl|NZ_CP026520.1_prot_WP_241688829.1_1742 [locus_tag=PC41400_RS08965] [protein=Imm64 family immunity protein] [protein_id=WP_241688829.1] [location=1973956..1974309] [gbkey=CDS]
MSADLVPYKDLNVEIEIEKCEEYSGFLVSFRESEVLPDYSEESLETATETIVKLVAEIYPTLQFDYAFCDLEAQIEFSPR
EFKPERYSLSFWPDLSNGELKVTKSSWHINGLTERQE
>lcl|NZ_CP026520.1_prot_WP_042231070.1_1743 [locus_tag=PC41400_RS08970] [protein=hypothetical protein] [protein_id=WP_042231070.1] [location=1974331..1974792] [gbkey=CDS]
MTKKPYKEIGSLSIHSVFEEQSGIIKVSMNEMGYVEVSISESYEHGMTAEIDLTFEEYERVLKAVDKANDILLSDIFHQE
RYGVDFIKVSGQSLKRQGFILVNVFEHSIILVVSLTRGADADIMIGKEEISLLVSLFKKVEMILDIKQFDEKA
>lcl|NZ_CP026520.1_prot_WP_042231069.1_1744 [locus_tag=PC41400_RS08975] [protein=hypothetical protein] [protein_id=WP_042231069.1] [location=1974833..1975195] [gbkey=CDS]
MKLVGSKTELDIREELINSKKSLFKGKSNNKLFDVLQVNFPKLKTAYTISWIPEQGEDLYKILIDDSIIVDIEINRVNRE
AIPIVKTISISEYSKGLSKISKIRLAVAMDLAQKDLKDPT
>lcl|NZ_CP026520.1_prot_WP_042231068.1_1745 [locus_tag=PC41400_RS08980] [protein=hypothetical protein] [protein_id=WP_042231068.1] [location=1975223..1975882] [gbkey=CDS]
MEYAKTVKAINHLFTYKIFDKQIYHKVQSLFSNLSVRDAAGIFGDLSNEAIFGLYLVLDELKTTNEAKEHIIYKYYGLKI
KEQANESVEGSLVEQILEDYKKTSFLALESLIVKMLKEDRISENQFEKLKRSFDSKIIEKEIYSNRIRSKIKGKFPLSES
ELLKLFEYENYKLIEDALAQELIECSALPLFRLPNKGDKNKKIKTVLFNKATKLIKMKP
>lcl|NZ_CP026520.1_prot_WP_042231067.1_1746 [locus_tag=PC41400_RS08985] [protein=hypothetical protein] [protein_id=WP_042231067.1] [location=1976046..1976303] [gbkey=CDS]
MTANFYKIQEIINEWNPIEIEPLLDDEYSFEVEYIVEFISEQKTGLTLLALRETINEVFNQEFERFYTQSEQTLDIAKKI
MHVCL
>lcl|NZ_CP026520.1_prot_WP_042231066.1_1747 [locus_tag=PC41400_RS08990] [protein=M4 family metallopeptidase] [protein_id=WP_042231066.1] [location=1976682..1978994] [gbkey=CDS]
MKKSLSVIVLSTIMALSAIPVGAASGNSASGSPADLAAPRFIGEKWIAPQNASLEAKVWGYLNAKKGLLNLAGDAEQSIR
ITDQATDAETGTQHVRLKQYIQGIPVFGTDQTLHFDKQGNVSAYLGSTVADSGQQSLTAISPKISAEKAIAIANKDMEKR
IGKLGAQEREQQAELNFYPVNGQNVLVYITEVNVLEPSPQRETYFIDAATGKIVNQYGMLDHATGTGVGVLGDTKTFTTT
QSGTQYVMQDTTRGGGIITYSAGNTQSLPGTLMRDADNVWTDPAAVDAHAYAAVVYDYFKNTFNRDSLDGRGMAIKSTVH
YGSRYNNAFWNGTQIAYGDGDGSTFRAFSGDLDVIGHELTHGITEKTAGLIYQGESGALNESISDVFGNTIQGTNWLIGD
DIYTPSIPGDALRSMENPTLFNQPDHYSNIYRGSDDNGGVHKNSGIPNKAFYLLAQGGTHRGVSVTGIGRGDAAKIVYKA
LTYYLTSTSNFAAMRQAAISSATDLFGANSAQVNSVKAAYAAVGIGSPPPAQDTQAPTAPGNLVSTGKTSSSVTLSWSAS
TDNVGVTGYDVYRGSVLAASVSGTSATVSGLSPSTAYTFTVKAKDAAGNVSPASSPVTVTTSSSTDPTDTLPPTAPGNLV
STAKTSSSVSLSWSASTDNVGVAGYDVYNGSTLATTVTGTSATVSGLQGGTSYTFTVKARDAAGNVSPASNAVTVTTDTG
SGAAAWAAYTTYQVNDLVTYQGKTYRALQGHTAMPGWEPANVPALWAVVS
>lcl|NZ_CP026520.1_prot_WP_053228737.1_1748 [locus_tag=PC41400_RS08995] [protein=hypothetical protein] [protein_id=WP_053228737.1] [location=1979260..1979634] [gbkey=CDS]
MIIKSKLTRWMAIIIVSLLGIVAVVLVIGLNTLKKQHEEEIKTVISKKGGIVLKIERVEPETSAFKNDFNKSNVIYRIIY
KNNVDSELLAWYRGINVPNDIHGKNPATLVGGFEEKWIFQSELK
>lcl|NZ_CP026520.1_prot_WP_084706626.1_1749 [locus_tag=PC41400_RS09000] [protein=hypothetical protein] [protein_id=WP_084706626.1] [location=1979866..1982136] [gbkey=CDS]
MAYVCSRNKLSSRLNRNNFKRFLQLSSRVWICSLLITMFIVEQVSAETNGTSTDYGYVYDKTGRLEFSILPSGRTVNYQY
DLSGNLLKKTYTPLTNSESPQLTLNSPNGVEVSPTLVYSTRPTITWTQTDDKNIFDAFNIKLYSLNSSKPVFDYYGRTND
GFWKLDRDLEVGQKYRVEVRVTDGEQWSNKSSGWFQVQKDSVEIPNSEFNGESLSFDGTKQLEFTDVAVNTDSGGKNTVE
FWMNWDGTENVMPFGWNGGYNLWFDNGYFGFNTGNADIYGIPSAQLKNKWVHVAAVFYNGVPNAANSELYINGVKQNISQ
VQGITASNVTATSTVYVSGSGVDKSKYKFTGRIADVHMWNRALTGSEIQKNMYGVLNGNEPGLVGNWKLTDPALTSLSFD
GTRQLEYTNVAVNTADGAKNTVEFWMNWDGTENVMPFSWNGGYDLWFDNGHFGFNTGNADLYGISSAGLKNKWVHVAAVF
YNGVPDATNSELYINGVKQNISQVQSITSSSVTATPTLYVSGYGVDKNYKFKGKIADVHVWNRALTGSQIQSNMYRTLNV
NEPGLVGNWRVSDPILKASSFDGTKQLEYTNVAVNTTSGAKNTVEFWMNWDGTENVMPFGWNGSYDLWFNNGYFGFNTGN
ADIYGIPSAELKNKWVHVAAVFYNGVPDATNSELYINGVKQSISQVQNVTTASVTVTPTAYVSGFGVDKANYKFKGKLAD
VHVWNRAITSSEIQSNMYRALMNNEPGLVGNWRLAD
>lcl|NZ_CP026520.1_prot_WP_053228736.1_1750 [locus_tag=PC41400_RS09005] [protein=GBS Bsp-like repeat-containing protein] [protein_id=WP_053228736.1] [location=1982557..1983312] [gbkey=CDS]
MTKLFKILSTICGVILLLMIMPSNTSAQAENIVEAPSRAYLFNGSYEIKVLNSNATRVRFPTWTLENGYDDLIWYEGEKM
EGNNWRVKIPLSDHNNETGSYVTHIYLTDSTGKENILVEKVVTIAEDPEPVIEYPTTVDISKGYYEIYVSNVFASKVAFP
TWTESDDQDDIIWHKGIPMGEGKWKVVIPFVKHNSEAGKYITHIYKYNDKDEQKFVGGGVTEVTGMVSYTKYIYDANGNL
IRKQFTWKEYK
>lcl|NZ_CP026520.1_prot_WP_129112464.1_1751 [locus_tag=PC41400_RS09010] [protein=hypothetical protein] [protein_id=WP_129112464.1] [location=1983425..1983808] [gbkey=CDS]
MNQELGTLTEGSYYLFLVSYKATSSTTVSMRVNDTQTNWGLNLGYKYFKTDDTWRRSHIKFQATGNPITIIGAYLDSGVS
KVQVDGARLYKISESIYNRIDNEEGYSGDLLYLKYPYNSFNEILGDT
>lcl|NZ_CP026520.1_prot_WP_053228735.1_1752 [locus_tag=PC41400_RS09015] [protein=RHS repeat-associated core domain-containing protein] [protein_id=WP_053228735.1] [location=1983918..1989146] [gbkey=CDS]
MKKKINYLVLFCLMVQMLAPLYVIHANEQSASNPQQISQPLLTVESLSKTYGVPETDLLSELNKGYSLLEIQSALQTRQD
PSQPLGEVLNQMNPTIEEQLKKEDYSAERFAEKQSQPGPTAPTVSEQTYGTYVESSGDSIIPDPSLLYPSGVTDATYGRV
SLLSSSNYPTSYDELAVKRLDIKADRAPWAFPISEDVSTLNGSLQIQTTDMTLPGRNGLSFELKRKYDSSDALYYEKDVL
NDSVYTTQYYPELTTRLYFKYSNEMVTEDGGATDFTFGPGYRSVIRFKNDYTYWYLPFEIPYVEEFNNEQYDSFKKAWPY
VDPENPNTQPFIQKEDIRLDGLEFIAKAYTTGNVRKEGKPQYIRTGYANKTKPIKNENRYPIGKGWSWDLPYIERKDNNK
TYIRLFGGATYELDDRTLKGYPWKDLTMYYDSSVRVNNLLSTFRLDSLDGKKQYFAHNGELIQITDAYQNTIQFEYQDVT
PYGQVLTKIKDAIGNEISIAYTEKEVTLTSGERTVKYDKIKDPQGNKELLSQVTDPAGRKTQYVYNVEAAPFDLVRDPKM
KDNYVALLKQVYYPTKARTDYTYEKFNRSLGYFGQETVHRVKAREDVVTFADGTESRSNPVDYTYSGDSGSVQNKNTSYT
TIVNNGRTQTSYSYDRVYIDDNTPEVFYNTQIKQDDGTTQSITMMEYNRTNRWPSPTKTTTKKVQGSSSSTEQVSQRTYD
EYGNVLTETDPANTVSTYQYDATTHLLSSVTAPATSGLNSYMELERYPGTNGIKEVRVKENNSAGALKAQSSYTYDAHGN
PISITIKDDAQDIVVNNGYGTQYHSAFPAEQSVQVTDAAKQNMTVTQRFEYNPSTGAMTKFTDGKGLMTKYEYDVLGRAT
KVIQPDSSTYTYTYNDAANEVTTVDPTGVTQITKWNPLGWKIKSGISGKASQEFGYDVYGRQTWSQDGAGNRTSYEYDKW
DRLIATTYPGAEQAKATVSYDDINRKVISKDGENNRTEETYDLLGRATQRDISSATGALIGSVKFTYDAAGKVLTKSDGA
GSNSGDVQTTQYSYDAMGRLASVTDAEANTTRYTYSKASNLIQIEYPDHNKKLKSYDQMGRVIQETDPLGQIETYFYDTN
SNLTKSIDRKGNIHTFTYNAVNLLTASETTDEKISFSYDAAGRRLSMQDGIGTTGYTYEPSTGWLTQVKYPDQRTLQYAY
DQQGKRTKMTDPFGVVTNYEYDTQNRITAVGTEPAEWDATYQYKKNGLLAATQAKNGVHAAFTYEGVNLTALTQAKSGAA
VQSFSYGYDLHANQTSKVENGVTQSFSYDALNRIATSSDYNENYSYDNRGNRLTLQSDKPINPVDTSYTYDNRNRLNAVQ
LDESSHVNYKYNGDNLLTERTENGITTRYYYDGDQMIAEGTVTNGVAAHKASYLRGNQLVARVDANGSKAYYVHNGHGDV
VELRDANGNVINQYAYDLWGNPTTVLKGVDNPFLYSGEYWDESVELQYLRARWYDPSMGRFINEDTYEGQLDNPLSLNLY
TYVENNPLTHVDPSGHCGIKSGGGLYNCNEVDTKLISLKQTAESPMITKSGKAAVASEAWRLRNSAGSYDEKNNPSGYTV
RYDKDLDYYMIYDVTDKLNNLMISYEKKYKDFSDNNSTKDVYLKFYNTVRGTAELDLKNQPDWQHSHFIYSGEYVDKDVP
GNISYGYFGKVMKIPDFMLLEAAGLAQMFAGTWKVEDYKSYFDDPRDQARIWQGIDLYKSWH
>lcl|NZ_CP026520.1_prot_WP_053228734.1_1753 [locus_tag=PC41400_RS09020] [protein=hypothetical protein] [protein_id=WP_053228734.1] [location=1989871..1990485] [gbkey=CDS]
MFGLKYILETYRMSNADLANKLEINRANISMWLKNKEIPAARIQDLKKIFPQIPKPLFNKELKKSEQLSIQQIYFTVTDQ
FESVRSQHIDSETGKEIWFTETVSENEGFISFLEHEHEKVSLLEKYEKLIEDNDMGNSRMNLLEDFASVIERNEPRSLEL
LDMVLYYLQDKHKSDFDSVFWNVGEPNETYKEFHAFLEKHKFIR
>lcl|NZ_CP026520.1_prot_WP_042231062.1_1754 [locus_tag=PC41400_RS09025] [protein=hypothetical protein] [protein_id=WP_042231062.1] [location=1990556..1990867] [gbkey=CDS]
MFDTIVMKASHVFIANEYWNNLKPVIKTFLDEETGLCRRSFVLHDEKIPYITYQEWSQSLIVQVSIPKFLYGNNVRLLQE
NDIFLFFQCLHERLFELFGVPLR
>lcl|NZ_CP026520.1_prot_WP_042231061.1_1755 [locus_tag=PC41400_RS09030] [protein=hypothetical protein] [protein_id=WP_042231061.1] [location=1990945..1991424] [gbkey=CDS]
METVAFKNKSSQIIFYNKQKEVIHSRESKEVIGQADGLLRMEIRPSHHDWTKFSRRKRVIDLLKVDVFTEITRKVMNQII
FPNEPDGISPAWLRSQPSKISQIETLLGFQLLQQEVPEGVLKQFYTAATYVNRKNLVKGIPLPKNQTKLELHIDYANLG
>lcl|NZ_CP026520.1_prot_WP_042231060.1_1756 [locus_tag=PC41400_RS09035] [protein=helix-turn-helix transcriptional regulator] [protein_id=WP_042231060.1] [location=complement(1991733..1991942)] [gbkey=CDS]
MIIKIRLREILAERGISQRQFALSMNMRLATVNHLCSDNVDRVYIRTLEQICEALQIDISELIVSQQEN
>lcl|NZ_CP026520.1_prot_WP_042231058.1_1757 [locus_tag=PC41400_RS09040] [protein=hypothetical protein] [protein_id=WP_042231058.1] [location=1992253..1992474] [gbkey=CDS]
MTIDDSLVLEEQRLKIQIETIEACQTAVFEVIRQYEDSACKLMQEEIVLAIHAIGLDLQTELLHVRFAQARQH
>lcl|NZ_CP026520.1_prot_WP_241688830.1_1758 [locus_tag=PC41400_RS09045] [protein=IS110 family transposase] [protein_id=WP_241688830.1] [location=1992662..1993903] [gbkey=CDS]
MNHIFVGVDLHKRHHTAVFLNGFKRKLGEFKFDNRPSAFPDLIKAAAKVSKGKGVIYGLEDTGGYGRPLAAFLVDHNQAV
KEVNPSLASAFRKANPIVHKSDSWDAECVGYVLVDQLEKLPDANPIDHYWAISQLVTTRNSLVREHIGLINQFHVQISYH
YPSYRKFFSQVDGKTALAFFERYPAPHHILGTSVEELREFLLKPSNFACSLKTAEKILKLIHSDGTTKRDYQEDRDYVVR
STIRRIQSNKKEIESLEHRIALLLPQTGYRLETMYGIDTVTAANFVAEIGDISRFGSSDKLARFAGIAPVLIGSGDKHKD
HKCRQGNRELHQLFYQLACRQIGVKRGSKEPNNPHFYAYYQRKQSEGKTKIQAIICLMRKLTNIIYSLMKNKSEYNMPSS
SNLQAGMSMGIKP
>lcl|NZ_CP026520.1_prot_WP_042231057.1_1759 [locus_tag=PC41400_RS09050] [protein=hypothetical protein] [protein_id=WP_042231057.1] [location=1993940..1994221] [gbkey=CDS]
MDIYIKQGGFQMLRKLEGLNAELFKTWVQEDDSTIVDIEGKHYLVKPLHNIVQEEIESDEELKMLIRQAKMDIAGNKTYT
TEEILEAIEKGDL
>lcl|NZ_CP026520.1_prot_WP_042231056.1_1760 [locus_tag=PC41400_RS09055] [protein=type II toxin-antitoxin system RelE/ParE family toxin] [protein_id=WP_042231056.1] [location=1994218..1994496] [gbkey=CDS]
MNIKWTQAARDSLKQIQSQHYTREETKRYQIELVRKIQHKILLLSTAIPTNEPSWAGTYRIFIDRYKVYYSFSADHQTVY
IEALRHQHQKSD
>lcl|NZ_CP026520.1_prot_WP_129112465.1_1761 [locus_tag=PC41400_RS09060] [protein=hypothetical protein] [protein_id=WP_129112465.1] [location=1994590..1995276] [gbkey=CDS]
MNLYTYVMNNPLTHVDPSGHIALDSIVWVLNKQLNSLKAEWERYNSSGDIKARDVASQKSKELRDNHMVLANSLYQSRRA
TVGILGANDTMLAYHDIVTDAGEEMTIAVNSMGTVFYEGDFVTYDYSEQVHIQSKSENLMGRAAKAAFGYYVGVIIGKKI
DPSKQVITHGSGLGGAWVTDNYLGSVPDVSDVKTMVYRTNKKTGTTENMIINTNSYNLKNWIYWKVYK
>lcl|NZ_CP026520.1_prot_WP_053228733.1_1762 [locus_tag=PC41400_RS09065] [protein=hypothetical protein] [protein_id=WP_053228733.1] [location=1996352..1996885] [gbkey=CDS]
MNIKKGSAPIFDLKLLVNEPLTRKELNKEELVFDLEKDCALAVTSMVCDIFEISGAVRFKVSGFGQKDWPVDCRTDLSTV
MEQVPEILKKIELNQYDFILDFYEQGMERQVAFSPDKDAIKLTCSSRTHWTPDPVFIHMKREQISRIFEDLYNRFLEVGE
ILCSDLIQDPLLEEWKV
>lcl|NZ_CP026520.1_prot_WP_129112466.1_1763 [locus_tag=PC41400_RS09070] [protein=hypothetical protein] [protein_id=WP_129112466.1] [location=1996905..1997330] [gbkey=CDS]
MEKSIKHGELLMHIRFLTLLKYAGVLAVLLIFNLIYRRLVNENLQVDNILYIFGTYMILGIVKYVRSEPMPSKTGHFLVF
FLFTLTVFIVDALLLYYENKFEITEPLVMSMIFLVFAFVISILAERISKLEDYESRKGEVA
>lcl|NZ_CP026520.1_prot_WP_042231052.1_1764 [locus_tag=PC41400_RS09075] [protein=hypothetical protein] [protein_id=WP_042231052.1] [location=1997327..1997668] [gbkey=CDS]
MKIIENLTVDGLVYTDENGENKKIDFKQCHEHWVQYRRRTENLTDEMTDEIRKKDFCIGQRDFTASPPYIEFFTKPFTRI
EFEISADEKDLQIELSRWKREIVSAGWTTIDLS
>lcl|NZ_CP026520.1_prot_WP_042231051.1_1765 [locus_tag=PC41400_RS09080] [protein=hypothetical protein] [protein_id=WP_042231051.1] [location=1997699..1997983] [gbkey=CDS]
MERTPKERLCWLLRLYKDKEIEAETFCDEFHMTYHYELNDEEVTETERVLFREIAAVAARFSPFEEDHQKYPGVYFTTED
VERVVRENSSELFT
>lcl|NZ_CP026520.1_prot_WP_241688831.1_1766 [locus_tag=PC41400_RS30230] [protein=hypothetical protein] [protein_id=WP_241688831.1] [location=1998041..1998412] [gbkey=CDS]
MKKALIDILLDPYSTEAEKDDAVIDLGEHFEDEETIAILLKVSNQDGIGEMVAASCGESLAFIWLRKSEINYNQLLLLKE
AALHEALSLIKAHRYDWFAEYERRLFSRREFENNDVQKDQQEG
>lcl|NZ_CP026520.1_prot_WP_042231050.1_1767 [locus_tag=PC41400_RS09090] [protein=hypothetical protein] [protein_id=WP_042231050.1] [location=1998437..1998892] [gbkey=CDS]
MHSFTYESKDSNNNSKEILGLLQFYTNLSRNVSWLIGDLELVPTYIGDYHPSGKVEPPQRVSLTSIQRFQEEGFLWLSLK
EIEEMLEDSLSVCRAVFVCLPEHMEVSHFHGDAESTSLQHESALHEITVLDDLIMIHSRSELLPPIRKGKV
>lcl|NZ_CP026520.1_prot_WP_042231048.1_1768 [locus_tag=PC41400_RS09100] [protein=PucR family transcriptional regulator] [protein_id=WP_042231048.1] [location=1999513..2001057] [gbkey=CDS]
MNLKGMTVRELLQLPILKDASLISGEEGLDRVVRYIDIMEVPDIKGWLREGELILTTGYSMRNDPGLVLELMEELRRADA
AALAIKPERFMGGIPAEMVELSNRYRIPVIQIPPGIPYIDITHTVMEQILNTQAALLRRSEEVNKTLTGLVLHNKGIQVV
ADSVAELVQAPIWVIHKHGRVIVSSPADAGDPPAGRTKSWDVSVDKQFAGRLIIGKDQLDELELVCIEHARLVFSLELMR
RKIAYDTEQRLQGNFFEELLMGLPLSRQDAENKGRQLGLQKDWCWEVCIVEGEPHLFEEHASLPAALGELTDSESAARKV
KSHLLRQGDRLVLLLASPAVPEPSTKQPVAGRTAGWADVLAPLLDSHSQIRTGFGGKAPLWEVYRSYVKAKKAMTIGCRL
DKERRTFSYDEIEMFDLLLESSAYVGFDAFTEKKIGKLSKYDRDNGTDLVPTFHAYLAAGGSLIEAANKLYIHRNSVKYR
LDRIKDIADIDMEDTSKRLIYYLCTSFYLLKNLE
>lcl|NZ_CP026520.1_prot_WP_042231190.1_1769 [locus_tag=PC41400_RS09105] [protein=aspartate/glutamate racemase family protein] [protein_id=WP_042231190.1] [location=2001219..2001878] [gbkey=CDS]
MIGVIRVLTTEDTKVLEQHGKLLEETYGLPVVSKCIPDQPLGIYDGETEAIAVPKIVELGKRMESEGCTVLVISCAADPA
IGPLRDSVSVPVIGAGSAAAYAALSMGRPVGVLGITEEAPAVIRDLLGDKLVRYTRPEGVTNTTDLMTPAGREKAVEAAR
SLLDAGTKVIVFACTGFSTIGLAAVLRSELHVPVVDAVEAEGLFASHYYKQLDDVREPN
>lcl|NZ_CP026520.1_prot_WP_042231047.1_1770 [locus_tag=PC41400_RS09110] [protein=DMT family transporter] [protein_id=WP_042231047.1] [location=2001888..2002811] [gbkey=CDS]
MTWTSARSYLLLIFCVFVWALNYMVRQVLLREFPPMFLSALSLTIVSGTFLLWAWGTRAFVKITGKDALWLCCSAGIGLI
ANQILLFRGLQLTTATNASLIFTLSPLFTAGLAALFLKERVTWRMVAGSVIAISGLAFALNVRGLQFNAGDWIMAGATFT
FSCNLIFVRILTRRLSPFIVTVYSFALSAVLFDPFVLAAGEATWIHPAGMWALVIVTVIVAQGLTGVMWNKGMQTLGAAK
AAIVLNLQPLMTMLLEFLLFGRAVTAPQLFGAALVFTGVLLGTVQLRIFSMKKKRTLPEVLPDSAGK
>lcl|NZ_CP026520.1_prot_WP_042231046.1_1771 [locus_tag=PC41400_RS09115] [protein=OPT/YSL family transporter] [protein_id=WP_042231046.1] [location=2003205..2004815] [gbkey=CDS]
MEEKEEKEKKHPSIFEPFALILNIVTSVIGAIIGMQLVTTLGVTPNTAIIGALVAMLIARIPVMIFRKYKSIHRQNLVQT
SISSATFGAANSLLIPIGIPFAMGATELIVPMLIGAALAMFVDATLLYKLFDSKLFPAKGTWAPGVATAESILAGDKGGK
RALLLGVGTVIGLIGSYFKISMSAFGVAFIGNIWALTMFGIGLMIRQYSVPWFDIDVNQLYIAHGVMIGAGIVALIQAGM
LLFKRPKTAGSEAESEEEVRDIRLTRTDKEARRTLSLGFAAYLVIALLIAIIGGIMTKMSLGMLIGFIIFAAFAAFVHEI
IVGIAAMHSGWFPAFAVAFITLLFGMLIGFPPVALALLAGFSAATGPAFADMGYDLKAGYILRGNGKDPELELHGRKQQY
ITAMIAFGVALLTVLLSYKGYFAQNLVPPIDKVYVSTIQAGASADIARQLLLWAIPGAIVQLLGGPSRQIGVLFATGLLI
VSPNACWAVLAGILIRIVVLKIFKKEAETPMSILAAGFIAGDALYSFFSSVFKLKK
>lcl|NZ_CP026520.1_prot_WP_042231045.1_1772 [locus_tag=PC41400_RS09120] [protein=DUF917 family protein] [protein_id=WP_042231045.1] [location=2004826..2005887] [gbkey=CDS]
MTRLKLNEEMVEQAVYGGAVLGGGGGGWIEEGLKLGRLALEVGEPELISLDEFGDDDLLVTVSMVGAPAAKEQFVKPVHY
AKALEMLAAKIGRPIQGLHTNENGAGTTVNGWFQSAITGVPLVDFACNGRAHPTGSMGSMNLSEVEDYVSHQAAAGGRGD
RYIELSISGSLDKASTMIRKASIEAGGLVAVARNPVTAAYARANGAAGAITQAIEVGRALLSAKGEAAIEAVVAQLGGKV
VTAGTVTDFSLETAGGFDAGTVRIDDAYEMTFWNEYMTLEKDGERFATFPDLIMTLDAKTAKPIVSAAIAKGQHIAVIAV
PAEKLLLSTTMRNRKLLQAVEDIIKKPVLSYLD
>lcl|NZ_CP026520.1_prot_WP_042231044.1_1773 [locus_tag=PC41400_RS09125] [protein=DUF1177 domain-containing protein] [protein_id=WP_042231044.1] [location=2005941..2006873] [gbkey=CDS]
MTLKQTMTVLDTLDSIYASGAKVKELFDAYPGISVTVSKVSGEKGSTDFVKVVIPGTDGKSKGGSAPTFGIIGRLGGIGA
RPSRIGLVSDADGAVAAVAAALKLADMQTKGDALPGDVIVTTHICPDAPTRPHEPVDFMDSPVDILTMNREEVVPEMDAI
LSIDTTKGNRVINHKGIAISPTVKEGYILRVSDDLLRIMEMTTGQHAVTFPVTMQDITPYGNDLYHINSILQPCVATDAP
VVGVAVTAQSVVPGCGTGASHETDIAAAARFAIEAAKETTSGTCAFYDPEEFSLITKLYGSMKILQTSGK
>lcl|NZ_CP026520.1_prot_WP_241688832.1_1774 [locus_tag=PC41400_RS09130] [protein=AroM family protein] [protein_id=WP_241688832.1] [location=2006960..2007631] [gbkey=CDS]
MGKLGMITIGQAPRTDVAPIIERCLEGRAELVQGGALDGLDKAHIERELFPEEGQYVLTSRLANGESVVMSREKLQPLLQ
KRISEFEKAGIRHILLLCTGVFPGLRAERSYLIEPDHLIPPAVKAMAGGRRLGVLVPLQEQCTTLAQKYEPYGLSPVFAV
ASPYKRDEAEFRRAAEELKEKSDIIVLDCMGYTEEARQFMAEASGKPVVLSNALMGKLVSEMI
>lcl|NZ_CP026520.1_prot_WP_042231042.1_1775 [locus_tag=PC41400_RS09135] [protein=aminopeptidase] [protein_id=WP_042231042.1] [location=2007661..2008608] [gbkey=CDS]
MNESLIEVGKHVLQSSLGLQARESFLVVSDEDKRHLAEPLYEAGKALGAEAVLMVMKDRYRPGEEPPSAVAAAMITSNVV
VCITKQSLTHTQARKNAAAAGARLATMPGITEDMFLHGAISADYAEVKRLTDQVTKLLTGAREVRIEKDGYTLTFPVEGR
KGVPSTGLYLNPGESGNLPSGEAYIAPLEGTAEGQILIDGSIAGIGKLSEPLLLTIEQGRIVKAEGEAGPQLLGMLGDKD
GRLLCEFGVGTNDKARITGVILEDEKVYGTIHVAFGSNNTFGGTITAGVHIDGLVKEPTVYLDGKLLMEKGKLAL
>lcl|NZ_CP026520.1_prot_WP_042231041.1_1776 [locus_tag=PC41400_RS09140] [protein=sensor histidine kinase] [protein_id=WP_042231041.1] [location=complement(2008732..2010000)] [gbkey=CDS]
MAQIKDLLLQLFFSLMPFVLYSFYYRDRDTNYTKKFILITSMICLFLSMTFASSVEKGIQFDVRYVILFFALVFGGFRIG
LLLMLEFVLYRLYLGGPGTSSAMGMMMVMFPLSVLLCSVYHRTSQKYLVAVMAGLCYSFFPLLLLYINRPDYILTHLTYH
ILIIPVHNSVGVLLLIYLFEKSATDKDMYLRYLQSEKIEAVGQVAASLVHEVRNPLTTVLGFLKLIRQDQAMPPEKLHRF
IDISIEEVKRTEAILSEYLSMSKPLTQKREVIDLAHELEAVAEVMTPYATMNNVSLILHVPSEPVRLSANRAEMKQLLVN
FVKNAVEACSAIQKGNVTLRMHTDGLLVKVHIRDNGIGMNKEQLRRLGSIYYSTKNTGTGLGLTYSYQVIRSMNGTVSVF
SELHKGTEFTLSLPLASKDPDG
>lcl|NZ_CP026520.1_prot_WP_042231040.1_1777 [locus_tag=PC41400_RS09145] [protein=TetR/AcrR family transcriptional regulator] [protein_id=WP_042231040.1] [location=2010173..2010766] [gbkey=CDS]
MSVNNPEKVTRQREERRSQILHASIGIFANKGLAAAKISDIAEAAGISHGHVYNYYSSKEELFVSLVKMLQERYGRVCEE
ARGREGNARDKLLWHARKIGEDENALNGMHVVLHAETIAGISAEEKREIPRRAVENLQPIIAIVKQGQEEGTFREGRPEE
LAVLYFALLNGVSGARRRGLGAVASPDEEQLVGLLCK
>lcl|NZ_CP026520.1_prot_WP_042231039.1_1778 [locus_tag=PC41400_RS09150] [protein=aliphatic sulfonate ABC transporter substrate-binding protein] [protein_id=WP_042231039.1] [location=2011061..2012077] [gbkey=CDS]
MNKLQASLAAAALAIGLAGLTGCGAGQAATTASGTDAPKGAYTVNLANNGGGGLFLLAKEKGWFEEEFAKIGGDVKWSEF
PSGPALLEALAAGRVDFSWLGDGAAISAQANNLPVTTLSLLSEGLKGLNIVIVPAKSTAQSLADLKGKKIAIAKGTTLHV
FFIKAVKQAGLKESDFQLIQLQAEEAVAAFESGAVDAWVGLDPYTSIQIAKNGARVLTSGEALNIKAPVLTIGRTDFVKE
HPGAAEAFLKGFEKAIAFQKEHRDEVVEFIAKAKKADRATIELVVQNTEYSNVPISKEIFGVQQASADILLEAGFIKKKI
DVTHNTDSKAIEAVKAAK
>lcl|NZ_CP026520.1_prot_WP_241688886.1_1779 [locus_tag=PC41400_RS09155] [protein=ABC transporter permease] [protein_id=WP_241688886.1] [location=2012208..2012963] [gbkey=CDS]
MKIALLGLVLPLAALAAWQTASGLKLVSATLFPSPLSIAEEFAGMVQSGELFHHLKVSVQRAAYGFLLGGSLGLLTGMLV
GLRKRAEYMLDPTLQMLRTVPHLAITPLFILWFGFDELSKILLIALGAFFPLYVNTFLGVRNVDSKLFDVAKVLEFSRFR
QITKLVLPSALPNVLLGLRLSLGVAWLGLVVAELMGSSEGIGYMIQDARQFSRTTAVFVGILIFAVVGKLVDSVVRLLEE
RLLKWRDNYKG
>lcl|NZ_CP026520.1_prot_WP_042231037.1_1780 [locus_tag=PC41400_RS09160] [protein=TauD/TfdA family dioxygenase] [protein_id=WP_042231037.1] [location=2013042..2013959] [gbkey=CDS]
MSEQVKVIPVSGRIGAIIEGISLSGNLDESTVGFIREKLLKHKVIFFRGQNHLDNAGQEEFTKLLGSPVAHPTIKPKDGT
DYVLELDSTHGGRANVWHTDVTFVDAYPAASILRGIVIPEAGGDTVWANTAAAYELLPAELRELADKLRVLHTNEFDYAA
TIRTVNALGSEAARKYRESFASTVYETEHPLVRVHPETGEKTLVLGGFAKKIEGFSVADSQKLIALFQEHITSLENTVRW
RWAEGDVAIWDNRATQHYAVNDYGDQHRVVRRVTIAGDVPIGVDGRKSATRIVSEPAPAAAASAS
>lcl|NZ_CP026520.1_prot_WP_042231035.1_1781 [locus_tag=PC41400_RS09165] [protein=SIMPL domain-containing protein] [protein_id=WP_042231035.1] [location=complement(2014339..2015013)] [gbkey=CDS]
MYGFAPPPSLPPSPGYRAGKPAVIEVTGDGIVSVIPDQAVIVLGAVTEGETLSPAQAENARIISAVIDSLLQLGIPRSSI
QTDEFRIDPQYDFQDGKQIFRGYRVTHLLKITTDRVGETGTIVDTAVSKGANSVSGISFRTSQPQKYSKEALTLAVRNAR
EKALTIAASLGVTLAAVPESVRELSGQNEPVPFKASMLAAGPATPIQPGELHVYAAVSARYTFS
>lcl|NZ_CP026520.1_prot_WP_042231033.1_1782 [locus_tag=PC41400_RS09170] [protein=AAA family ATPase] [protein_id=WP_042231033.1] [location=2015216..2015776] [gbkey=CDS]
MIRDIHAMFLVTGIMASGKSTVAQLLAERFEKSVHLRGDHFRKMIVNDRKEVQPDAEADQMEQLLLRYRLTAQAADTYYS
AGFTVVMQDVVVGRVLSDFLSLLEDRPLYVIVLCPGEEEVASREAARSKKGYGVWTVSALNGVLHGETPRVGLWLDSTGL
TPEETVDAILERVWEEGIVRTQDFGD
>lcl|NZ_CP026520.1_prot_WP_042231032.1_1783 [locus_tag=PC41400_RS09175] [protein=phosphotransferase] [protein_id=WP_042231032.1] [location=2015941..2016732] [gbkey=CDS]
MSNRSNEELLAGGNVSKVYRSGNTVRRELKPNSPNIHALLNHLDNKGYGYAPKYLGTDEKGREILSFIEGEAGNYPLKKY
MRSDDVLQEIAKMLRLYHDAVSDFSFDDSWQPFDHTPQPLEVICHNDFAVYNIIFQHEKPIGIIDFDAAGPGPRLWDIAY
TLYTCVPLSRLHLTETGEIVYYDSFQDASRIKNRVNLFFDCYGEGFAENYIDMVLLRLEGLCKTITRRASEGDLAFQKML
DEGHLEHYRKDIHFISEHRNDWT
>lcl|NZ_CP026520.1_prot_WP_042231031.1_1784 [locus_tag=PC41400_RS09180] [protein=GNAT family N-acetyltransferase] [protein_id=WP_042231031.1] [location=2016774..2017298] [gbkey=CDS]
MSTSREAEAWAGADLEIRKAVPAETPLLQELYVEAARWRVSRGILLWDEKTFTPEYFAELMADCELFIAYSGGEAVGCFT
LQWSDKLLWGERDRGDAGYLHRLVVTRKRSGLRLGTRLLEWAEDYIRQTGKTYFRLDCMSENARLNAYYRNAGFVYLETV
TGDGWSANLYEKKL
>lcl|NZ_CP026520.1_prot_WP_042231030.1_1785 [locus_tag=PC41400_RS09185] [protein=GNAT family N-acetyltransferase] [protein_id=WP_042231030.1] [location=2017596..2018135] [gbkey=CDS]
MGSSPSIVIREAAEADREAVRAVMLEAYLQYADVMSAQRWQQYREDIGAAADKGDPVARLVAEADGKIVGTVQLFVSSEA
AYGRPELEIHTPIIRYLSVSPEARGRGVATELIKESAKRSLELGASTLHLHTSDMMASAVSLYERLGFQRAFDKEFHNGE
VLVKSYRLELRESAFFTAS
>lcl|NZ_CP026520.1_prot_WP_042231029.1_1786 [locus_tag=PC41400_RS09190] [protein=transporter substrate-binding domain-containing protein] [protein_id=WP_042231029.1] [location=2018237..2019106] [gbkey=CDS]
MKKQVNKWLAGAGILVLAVVSAGCGKAVETGGGTTTAATDKPGESPAGQARKVIVGTGTKFPQVCFIDQNGKLTGFDVEL
IKELDKRIPEYEFEFKTMEFQSLLLSLETNKIDFIAHQMEKNPEREQKYLFNKEPYSIFLNKVAVEINNDTIKTIDDLKG
KKVRTGATSNEAYILQEYNKKNNNAIEIVYDNATANDLVDKIRKGRIDATVTSDFALRNYTDKDGKVALKTVGEPLTESD
VLYVFRKTDGKLADRIDQAIKEVKADGTLSKLSIQWLGQDFSNVGTQKK
>lcl|NZ_CP026520.1_prot_WP_042231028.1_1787 [locus_tag=PC41400_RS09195] [protein=amino acid ABC transporter permease] [protein_id=WP_042231028.1] [location=2019140..2019922] [gbkey=CDS]
MGRRFEADYIFFFIPKLLSYLHITLLIVGSSILLGLIIGFLIALPRLYNVPVLRRISGVYVSFFRGTPILIQLFLIYYGL
PELLKIVQVDVSKTPVLLFVILTYSLHTGAYVSEMIRSAVAAVDRGQVEAAYAVGMNGYQAFTRIVMPQALGIALPVFGN
LVIGNLKDSSLAFTLGVMEMTGKSETLGTATQHFIEIYIALSLIYFVVSFILQKGFDAAEKRLLRHSRPLADTSGKTAAR
PNIFRRLQPGIKKEAPGYEP
>lcl|NZ_CP026520.1_prot_WP_042231027.1_1788 [locus_tag=PC41400_RS09200] [protein=amino acid ABC transporter permease] [protein_id=WP_042231027.1] [location=2019912..2020622] [gbkey=CDS]
MNLDPSFIGTAFLSLLSALPVTLYITAVSVLGGLLIGTVIALVRIYRVPVLYPAAVAYVTFIRGTPMLMHLFLVYFGLPA
VLDTLADRYGWSFNSGSIPLVAFVIIAFSITAGAYMSEVVRAGIGAVDRGQIEAAYAVGLTTAQALRRIVFPQAFAVSLP
NLTNSVISMLHATTLAFTISVVEINAQANIVASTNWKFFEAYIAAALLFWGITVLVEKASSALEKRFTLYNRGGVA
>lcl|NZ_CP026520.1_prot_WP_042231026.1_1789 [locus_tag=PC41400_RS09205] [protein=amino acid ABC transporter ATP-binding protein] [protein_id=WP_042231026.1] [location=2020619..2021371] [gbkey=CDS]
MIKLTGISKSFGKQEVLKGIDLTVHKGEVVVILGPSGSGKTTLLRCINFLEKPNAGEVTIGDFTVDCKRSGRKEIHTLRK
KTAMVFQQYNLFKHKTVLENVMEGLVVAQKLPKDEARARSAEVLLKVGLSGKLDAYPSQLSGGQQQRVGIARALALNPEV
ILFDEPTSALDPELVGEVLAVIRKIAKEGITMIVVTHEMGFAREVSNHVVFMDGGAIVEEGPPGELFNAPKEERTRQFLK
RITPELNYSI
>lcl|NZ_CP026520.1_prot_WP_042231025.1_1790 [locus_tag=PC41400_RS09210] [protein=LLM class flavin-dependent oxidoreductase] [protein_id=WP_042231025.1] [location=2021502..2022563] [gbkey=CDS]
MAIKLSILDQTPIYEGDTGTEAFNRTVALARRAEELGYHRFWVAEHHDSAHVAGSSPEVLISHLLAKTERIRLGSGGVML
QHYSPYKVAENFNVLSALGPGRIDLGIGRAPGGLPRSTQALQKNVVDADSLTDKIVELRKFLHNRLEEDHPLAGLRANPV
PERPADLYVLGASTGSARIAAELGLPYVFSLFINGDPQTAVEAIGTYRREFNYAGGAQPAAILALSVIAAGSEEEAEELA
GDHKLVKIHLESGRTITVSTLEHAEEFGKQSSENFTIEVKEPSVTKGTIETVKKELLELQRVTGVDEFIVLTAVKDFAKR
LRSFELLIEALSGADTAAEPEAETAGRRVSVEA
>lcl|NZ_CP026520.1_prot_WP_042231024.1_1791 [locus_tag=PC41400_RS09215] [protein=amidohydrolase] [protein_id=WP_042231024.1] [location=2022663..2023856] [gbkey=CDS]
MTSETLAQQEQLEELAAKLVAARRHLHENPELSHEEFETTAFIKAWLEEAGIRIAPYSLRTGLVAEVGGLRPGPVVAIRA
DIDALPIREETGLPYASKVPGKMHACGHDFHTAAVLGAAYLLKQKEEELPGTVRFLFQPAEEKASGALKVIGSGALENVR
AVFGLHNKPDLPVGTLGIKEGPLMAAADGFVAEIEGRGSHAALPEAGSDPIVASAQIVSAVQSIVSRNISSLDSAVVSVT
KLHSGTAWNVIPEKALLEGTIRTFDEGVRSRVLARFREVVEGVAAASGTKASLRWIQGPPPVNNSAELAELARTTADALG
YSAVTPLPSPAGEDFAFYQREVPGLFVFVGTDGPHEWHHQAFDLDEAALPVSARFFSELALRALLELADPPAGGANG
>lcl|NZ_CP026520.1_prot_WP_042231023.1_1792 [gene=solA] [locus_tag=PC41400_RS09220] [protein=N-methyl-L-tryptophan oxidase] [protein_id=WP_042231023.1] [location=2023853..2025013] [gbkey=CDS]
MNGGMTYDVIVVGAGSMGMSAGYHLAKQGVRTLLIDSFNPPHGEGSHHGEPRLIRHAYSGGPAYITMALRAQELWEELQA
ETGTELMVPSGVINIAGADAFRTRAEDARKLGVSVELLDAAEIGRRWPGFELPEHFAGMYEPDAGYLFSEKCVAAYRRLA
EEAGARLLTDTPVRQLNAREGGVAVHTDRGVYYAQQAILATGAWFRTLEPFVKLPVRAVRKAVGWFQPKQPLFGAGRFPG
FTIGGGDGGYYGFPDIGGAGVKIGRHDTGVEWTAGEALEPFGRYEEDEGDLRRALEAYMPQAAGPLNRGAVCKYELSPDE
QFMIDRHPVHPSVLLAGGFSGHGFKFSSVVGEILADLMLYGRTPFDIGPFALTRFTQGSPGPMVSG
>lcl|NZ_CP026520.1_prot_WP_042231022.1_1793 [locus_tag=PC41400_RS09225] [protein=DinB family protein] [protein_id=WP_042231022.1] [location=2025034..2025516] [gbkey=CDS]
MSGSGILNVDIQLYLSVYDQLKLAVAGLDEEQLKRKEAPGKWSITEVLAHLADHNIVVSFRIREILAGSDKILPGFAQDP
WVEGQRANESGVSDLLDAYAALLQYNAVLFRRLTEADWEKTGVNFKGETVTLASVVNAFTAHVQTHLGQIERIKTVVAAV
>lcl|NZ_CP026520.1_prot_WP_042231021.1_1794 [locus_tag=PC41400_RS09230] [protein=LLM class flavin-dependent oxidoreductase] [protein_id=WP_042231021.1] [location=2025541..2026866] [gbkey=CDS]
MSKKRQLKLGALLHGVGGNPAMWRHPDSQPDASVNFELYKKWVRKAEAGKFDLIFIADGLYINEKSIPHFLNRFEPITIL
SALAAVSTHIGLVGTLSTSYSEPFTVARQFGSLDAISGGRAGWNVVTSPLEGSALNYGKSHPEHDLRYKIAEEYLDVTKG
LWDSWEDDAFVRDKESGVFFDPGKLHTLNHQGEFFSVRGPLNIARSRQGQPVIFQAGSSEVGKNYAAKVADAIFTGQENL
EDAVRFYKEVKERAVSHGRSKDDVLIFPGIGPIIGSTEEEAERKYQEVANLVTIETALDYLGRFFEHHDFSQYPLDEPFP
DLGDLGKNSFQSGTDKIKEEAKEKNLTLRQVALQAATPKGHFIGTPEKVADLIQEYFEAGAVDGFMLSAAVPSGLEDFVD
HVVPVLQQRGIFRTEYESDTLRGNLGLPVPANRYAKQPETV
>lcl|NZ_CP026520.1_prot_WP_042231020.1_1795 [locus_tag=PC41400_RS09235] [protein=OsmC family protein] [protein_id=WP_042231020.1] [location=2027059..2027622] [gbkey=CDS]
MSQLNDYLAAKKKSLEEREEERERVANQPSDVLKAKVRAKDRSGVREIRIRNFQILSDSLPDFAGFDLGPNSAEIQLGVL
GSCLTHITLIQAALRGVPLESLEVEVEGELHKFAGKDGYEEIPFWPHNIRYTLHIESAASKEEIAELHRGVEEVCPILNL
LQRPQEITGKVLHSSGSGEAGEVTVGS
>lcl|NZ_CP026520.1_prot_WP_042231018.1_1796 [gene=cysK] [locus_tag=PC41400_RS09240] [protein=cysteine synthase A] [protein_id=WP_042231018.1] [location=2027658..2028623] [gbkey=CDS]
MSARDGEEVEDAGHSPAMAGSVLELIGDTPVVRIRRLSGTNDAELYVKLESFNPGGSVKDRAARHMIEQAERDGRLLPGG
TIIEVTSGNTGIGMALVAAVKGYRLIVIMSDNMSAERTGILQAYGAEVVLYPAEEKIEGGLRRARELERRIPGSFVPVQF
ENAGNADVHRFTTASEIWEQMQGKLDAFVCTSGTGGTITGTGESLKSRLPDLRIYVVESRCSPLLSGGEPGTHGIPGISP
SFIPPVLNTQLYDEVLQIGDEDAYETARELARREGLLLGPSSGAAVWAGIRIAERLGRGKKVLCIAPDTGERYLSTGLYD
A
>lcl|NZ_CP026520.1_prot_WP_042231017.1_1797 [locus_tag=PC41400_RS09245] [protein=LLM class flavin-dependent oxidoreductase] [protein_id=WP_042231017.1] [location=complement(2028876..2030222)] [gbkey=CDS]
MAANRQIKLSAYLVGTGMHVASWKHPNAQRSASISVDYYKALAQTAERGKFDIAFIADSLAINEDSHPNILNRFEPVTLI
TALAGATSKIGVVATASTSYIEPFNLARSLLSVDQISKGRAGWNIVTTRDLSGSTARNFGAQDHLEHGFRYKRAEEFVEV
VTGLWDSWEDDAFVQDKESGVFFHPDKMHRLNHRGEFFSVEGPLNIARSPQGYPVLVQAGTSESGQQFAAKRAEVIFSIK
YSKQEAREFYDTFKSKVASFGRSPDEVVILQGIFPVIGRTQEEADEKYRQLESYISEETGFSFLKDYFPGVDFAGLTLES
RAADAGLDRLQLDKSDYRKHQPVIAGENPTLREVYSLLTGSFSREDWVGTPEKIADTLEEWFTERAADGFMLMAPLLPDG
LHDFVEEVVPLLQQRGLFRTEYAGDTLREHLGLRVPENRYAREEVSRP
>lcl|NZ_CP026520.1_prot_WP_042231015.1_1798 [locus_tag=PC41400_RS09250] [protein=ATP-binding cassette domain-containing protein] [protein_id=WP_042231015.1] [location=2030498..2031283] [gbkey=CDS]
MILVENLTKSYKSAGGPVTALENINLRIQKGDIFGIIGFSGAGKSTLIRCLNRLEEPDSGTVEIEGLRITDLDEKELRMA
RRKIGMIFQQFNLLDSKTVFQNVAFPLKVAGVPKPQLAERVREILDLVQLGDKENVYPSQLSGGQKQRVGIARALVNEPD
VLLSDEATSALDPQTTYSTLELLKDINRQLNLTIVLITHELDVLQHICRNMAVIEKGRIVETGTVDKFFLQPESDTARRF
VSIIDHYRRERTVLEGAGAGI
>lcl|NZ_CP026520.1_prot_WP_042231014.1_1799 [locus_tag=PC41400_RS09255] [protein=methionine ABC transporter permease] [protein_id=WP_042231014.1] [location=2031280..2031948] [gbkey=CDS]
MRDDLVDLLVEGLQETLFMVVWASLFAFLLGSALGITLVVTEKGGILAAPKLNKVIGTAINAVRSLPFIILIVLLLPLAR
LIVGTTLGPTAAIVSLSIGAAPFLGRIIENSLKEVSSGKIEAAKSVGASPWTIILRVLIPEALPALIRAVTIGVIAITEF
TAVAGAIGAGGLGSLAIRFGYQRFREDILIATVILIILIVQIIQWSGDSIAKSVIRKRFKME
>lcl|NZ_CP026520.1_prot_WP_042231013.1_1800 [locus_tag=PC41400_RS09260] [protein=MetQ/NlpA family ABC transporter substrate-binding protein] [protein_id=WP_042231013.1] [location=2032005..2032871] [gbkey=CDS]
MAHTRLSALTAILLAVTLTAAGCGSQGKGEVTASEKTGSTPAASKEPVKEVKLTVGAAAVPHAEILEVVKPKLKEQGISL
DIKVFDDEGQLNPALKDKQIDANYFQHVPYLDSVKKEKGFDFAVTAKVHVEPIGFYSSKLKSKDEIKEGAKIGIPNNPSN
EYRALVLLQQQGLIKLKEGLTTYEATPKDIVENPKNLKFFETDSATLPRALPDLDGAIINTNLVLEAKLDPKSALFREDA
NSPYANVIVVRKGDESREEIKKLDAALTGPEVKKFIQDKYGVAVVPAF
>lcl|NZ_CP026520.1_prot_WP_042231012.1_1801 [locus_tag=PC41400_RS09265] [protein=hypothetical protein] [protein_id=WP_042231012.1] [location=2033250..2033735] [gbkey=CDS]
MDRRTTGDTARDNRTRGNRPADHRTSGNRTGGSRTFGHRTAGGRSDGNRNEGNRTLGNRTEGRRDDYGWQNPQNLPGGIW
PSPWNAPADRPEGEFAREDRTRDRHTGGSRTRGNRTEGNRTGGNRTEGGRAGGNRTGGRRTHGFPARLVAPVFTTKPTQK
R
>lcl|NZ_CP026520.1_prot_WP_042231011.1_1802 [locus_tag=PC41400_RS09270] [protein=CotH kinase family protein] [protein_id=WP_042231011.1] [location=2033856..2034911] [gbkey=CDS]
MSLPHFQLAIGTGGDRKLRQNVWSRRFVPAILSDGSSSVPIRIGYRGGHTREYAKKSYEIRIKGRTYHLNAEYDDPSLIR
NALSFRFFRMIGVPSPHTRHVVLSINGRNEGVYVLIEAVKRSFFRRRGILMKSLLYASNDSANFSLKDPDTRKRKKTLFA
GYDLMIGTGRDRDKMKAFIQSMHALKGDRLKAFLGKHVDLDNYLRWLAGAVMTGNYDGFEHNYAIYEHKPTGKYRMIPWD
YEGSWGRNCYGKRVNSNLVRITGYNDLTEKVLESKEVRQRYKALLKHLTGTAFTKDRIMPVVYEMHERIARDVYKDPGRQ
KSLSLFRSEPEVIGQYIEDRRKDIQEEIRLL
>lcl|NZ_CP026520.1_prot_WP_042231010.1_1803 [locus_tag=PC41400_RS09275] [protein=substrate-binding domain-containing protein] [protein_id=WP_042231010.1] [location=complement(2034926..2035936)] [gbkey=CDS]
MITIYDIAEKANVSAMTVSRVINNSGRISEATRKRVRKVMEELNYIPNSMARGLVLQKTKLLSLLITDITNPFYTTIARG
AEDTAKRSGYKLLFGNSDEDYVKEKDYVEMILSTRVDGVLFAPTGDASLEQLRQLQKHGIPFVILDREVPGIETDVVLGD
SKDGARKLVEHLIGLGHRRIALVNGSPDVSTARLRYTGYREAHLLGGLPLDESLVVSLNYREFQDEASLQALLSLPSPPT
AIFAANNLLAVGIIGALRQRGLSVPEDVSVVCFDDFGPAETVNPFLTVASQPAYQFGALGMQLLIERIENGPGSENRKIM
LPSELLIRSSTRTWDR
>lcl|NZ_CP026520.1_prot_WP_042231009.1_1804 [gene=fucU] [locus_tag=PC41400_RS09280] [protein=L-fucose mutarotase] [protein_id=WP_042231009.1] [location=2036075..2036500] [gbkey=CDS]
MLKGIPSILSPELLKVLMEMGHGDEIVLGDGNFPAASHAKRLIRCDGHGVPELLEAVLALFPLDTYGDQPAQLMAVVPGD
PVQPVIWETYGEVIGRHLPGVPVGQLERFAFYKRAKEAYAIVATSESALYANVILRKGVVT
>lcl|NZ_CP026520.1_prot_WP_042231008.1_1805 [locus_tag=PC41400_RS09285] [protein=L-fucose isomerase] [protein_id=WP_042231008.1] [location=2036560..2038350] [gbkey=CDS]
MNNREHRWSRPFPKIGIRPTIDGRRKGVRESLEEQTMNMAKAAAALLSGNLRYPDGTPVECVIADTCIGGVAEAAAAADK
FARSGVGVSVTVTPCWCYGTETMDMNASIPNAIWGFNGTERPGAVYLAAVLSAHAQKGIPAFGIYGKDVQDAGAAGIPED
VREKLLRFARAALAVAVMKGTSYLSVGSVSMGIAGSIVSEELFQNYLGMRNEYVDMSELVRRMEEEIYDKEEYGLALAWV
KDNCKEGPDNNPAAIQTSRERKDKDWETVVKMALIVRDLMTGNPKLAELGFGEEALGHNAILSGFQGQRHWTDHFPNGDF
MEAILNSSFDWNGRRAPYLVATENDSLNGVAMLMGYLLTHTAQIFADVRTYWSPESVNRVTGYTLEGAAEHGILHLINSG
PAALDGTGQQSRDGAPVMKPYWEITEEEVDKCLGETSWRPAAVEYFRGGGYSSDFLTKGGMPVTMTRINLVKGLGPVLQL
AEGYTVDLPDHVHDALDKRTDPTWPTTWFAPILTGEGSFRDVYSVMDNWGSNHGSISYGHIGADLITLAAMLRIPVSMHN
VDEKNIFRPRVWSSFGTRDEEAADFRACANFGPLYN
>lcl|NZ_CP026520.1_prot_WP_042231006.1_1806 [locus_tag=PC41400_RS09290] [protein=rhamnulokinase family protein] [protein_id=WP_042231006.1] [location=2038377..2039903] [gbkey=CDS]
MSVEKNGLHMLAVDFGASSGRTVLGTYNGSRLAVEEIHRFSNDPVRLGGSLHWDFLRLFHEMLQGIRACKPFTAGAPASL
AVDTWGVDYGLVDGRGRLLGNPYHYRDSRNEPAMRRVLNRIPEEELYRRTGIQTQSINTLFQLEAEGEAASQAGEDVRML
FMPDLFRYYLCGEISAEYTIASTSGLLDPQLRGWDEEVLALLPHPRAMFPSIVMPGTTAGSLRAALAEELQMSSFPVVAA
ASHDTASAVVSVPAQAGGSYAFISCGTWSLMGVETEQPVITEQSRKLAFTNEGGADGRIRLQKNIMGLWLLQECRRYWEL
EGETLTFGDMQAMAAGEEAAAAYVDPEDPVFLAPGNMPERIRSRCRQTGQPVPQTKSQMIRCIVESLAMKFSQTLHEIEE
LLGRRLDAVHMVGGGIRNRLLCQLTANITGRCVIAGPVEATAVGSLLMQAKARGELHTRDEMRQVVLASFPPDLYSPEDT
GRWQEAYARYRKINANAHHSERSVNEND
>lcl|NZ_CP026520.1_prot_WP_042231005.1_1807 [locus_tag=PC41400_RS09295] [protein=class II aldolase/adducin family protein] [protein_id=WP_042231005.1] [location=2039896..2040558] [gbkey=CDS]
MTERKIREELTKYAAKSVAKGLVVGPGGNISARSGEFMYLSPSGFALEDIEPHQWVPVHIPTGEIGDTDFRPSSEVLMHL
FCYRANPLLQAIVHTHPAYCIALTLVDDELPMLFPDQAALVGDVAFIPYVVPTTDKLAHLVAEKADRYSSMLLKNHGLVT
TGKNLREAYYRTEVMEESAKIYMIAKSVGEPRRLTEEEYKEIQGLESEAYRVQLLQNMKE
>lcl|NZ_CP026520.1_prot_WP_042231004.1_1808 [locus_tag=PC41400_RS09300] [protein=glycosyl hydrolase] [protein_id=WP_042231004.1] [location=2040921..2043983] [gbkey=CDS]
MKKLFAGLLALILLVSLLPERGTSAFNGEVALGSGSYSTVLPPGSVDTQSMIYKTANVTGAMPTNDWWSNLAWDSFSEAQ
YPHPLAMKNGSGGIRIYYPGNRITANSSCVCGWINDIHDFTVGHSGVAAFPDAKVDGYSDWFVKSQYKSGASEMNVSYGH
GSPYVYFTYAGGNPKLSFYDTPAVWSGGANTPVLGITVAGAHYGLFGANGSTWSGIGSGTLTNNGSSYFSVAVLPDPSQS
TLNKFAQYAYSHVTGTQVSYSYNAAASEVTSTFTFTTQAKQGTQSGTIFALYPHQWKNSPTPLLSYTYGSVRGLMKTGEG
SSFQTRMKFNGVLPALPDKGSYDRGQLQRYIDQAEAETYTGDGDTYWIGKRLGKLASLAPIADQIGDTTAANKFRTEIKS
ILQNWFKASDGNGNIKSGNVFYYNRNWGTVIGYPASYGSNNELNDHHFHYGYFIKAAAEVARVDKAWASAWGPMVNLLIR
DIAGSSRTDTMFPYLRNFDPYAGHSWASGHARFGDGNNNESSSEGMNAWAGMILWGQATGDTAIRDTGISLYTTEMNAIN
EYWFDVTGANRPAGFTRSTASMVWGGKTVGDGTWWTGNPEEVHGINWLPFTGASLYLTQYPDYAAKNYNALVSENGGTNF
DAWEDLVYMYRAISNPAEAKSFWSSRGNALSAEAGNSKAFAYHWIYNLDAMGVQDRTVTANTPLYAVFVKNGVRTYTAYN
AGSSAIRVTFSDGKSMSVPANGSNTEGAGGSTPTPTPTPTPTPTPTPTPTPTPTPTPTPTPTPTPTPTPTPTPTPTPAPS
SAPIPGKIEAESYASMSGVQTETTTDTGGGLNVGYLDTGDWMDYSVNVAKAGTYTLQARVASPNGTGQFQLRSGSNTLAT
VNVPNTGGWQTWTTVSAQVQLAAGTQKLRVAISGPQFNLNWLQFASADNVLTYTASDYTATVTKSGASETVVFAPKTSAA
YVDIHYKVNGANQQNVRMSASGGSWTHTIQGLSAGQQVELWFTYEKSGPQYDSAHYTYTH
>lcl|NZ_CP026520.1_prot_WP_042231003.1_1809 [locus_tag=PC41400_RS09305] [protein=DinB family protein] [protein_id=WP_042231003.1] [location=2044147..2044677] [gbkey=CDS]
MIFSELEPFLDKKRRELLEALDAFGPEERHLAEYEGGWNPVQIVEHLSIVEEQVIAEIGRLLNGEAVRPSEPEERKLVDI
TALFEQKGLLGSPKKSPPSAFPTGKVAYEDGLERLAEVRGNLKAFLPLLAARETNRLMSCHPLGVELNACQWLQFSAFHE
WGHIHQLKRIRRAHSG
>lcl|NZ_CP026520.1_prot_WP_042231002.1_1810 [locus_tag=PC41400_RS09310] [protein=hypothetical protein] [protein_id=WP_042231002.1] [location=complement(2044940..2045140)] [gbkey=CDS]
MKPVRNRSNSGLKRRRSAKSAGKVLAQRDSSGPIYSIGPSVDGKEQSFQLVLEKEAVYETPVYLQG
>lcl|NZ_CP026520.1_prot_WP_042231001.1_1811 [locus_tag=PC41400_RS09315] [protein=nucleotide-binding protein] [protein_id=WP_042231001.1] [location=complement(2045142..2046149)] [gbkey=CDS]
MDQQRKPKVFIGSSREAIPIAEGVLENLSPAAQANPWFAGAFNPGRYTMEDLDKQVQNNDFALFILSTDDVVQIRGKQYA
SARDNTIFEMGLFMSRLGRERVFFLMPDHIPESVNETAVGGLRTPSDLFGVNALTFEIRSEADEWVQATAAACSTVKRKM
RELGCLSEQPTINQLARILRLFRTLLKGVPFKSEEENSLQTLSEGIRISYTCPPQYTVKGVTIHRAEQDEWIKQVAGTSN
IGTMNRTYPLTANENLVPGDKRILVVDAHLNNRVTMNLHSSSIENEYLICYPLAKKYVLTVHMIGQDTLSSEQLNKIAGL
NGQLISSVNDLLGGE
>lcl|NZ_CP026520.1_prot_WP_042231000.1_1812 [locus_tag=PC41400_RS09320] [protein=GNAT family N-acetyltransferase] [protein_id=WP_042231000.1] [location=complement(2046276..2046827)] [gbkey=CDS]
MKIITTAQWDEKLWREAEPLYNEAFPDHGRKPIRIIRQMFQRKLCFLHLGIVENTVAVMAITGELPSVRALLIDYIAVRH
DCRGQGTGRHFIEELKHWTRRERGLSGILIEAEAGDGPGHRSRRRFWEACGFHLTDYVHEYSWVPERYQAMYVNLSPEGK
LADDGRLLFESITDFHKRSYQRT
>lcl|NZ_CP026520.1_prot_WP_042230999.1_1813 [locus_tag=PC41400_RS09325] [protein=Gfo/Idh/MocA family oxidoreductase] [protein_id=WP_042230999.1] [location=complement(2047040..2047963)] [gbkey=CDS]
MRIGIIGLGDIARKAYLPVITARNDVELIFCTRNTATLEQLASVYRISETVTEVGELISRGIRAAFVHSSTESHAEITKQ
LLQNGVHVYVDKPIAYTYEESLELTELAEKNGLILMVGFNRRFAPLYVTVKEKSSPNLILMQKNRPHSPDFARRFILDDF
IHVVDTLRYLVPGAVRDINVTSHRQDGKLYHVMLQLSGDGYTAVGIMNRDSGASEETVETIGGGRKWSVKDLNTAVELAD
GGERHYKFNDWDPVLHRRGFHAIIDHFIQTVKEGGQPLQSTRDALETHRVCEEIVKELEKQGVSAWG
>lcl|NZ_CP026520.1_prot_WP_042230998.1_1814 [locus_tag=PC41400_RS09330] [protein=rhodanese-related sulfurtransferase] [protein_id=WP_042230998.1] [location=complement(2048133..2049032)] [gbkey=CDS]
MNDTQQPYRILLYYKFAPIANPEEFAAEHREYCKNLGLKGRILIAAEGINGTVSGTVEQTEQYMSDMHANPLFADMVFKT
DPAEEHAFKKMFVRFKQELVTFRYDEELDPNVISGKRLSPAEFHEQLQNEDVIVLDGRSNYEFDLGHFRNAIRPDVEAFR
EFPEWIRENMADLKDKTILTYCTGGIRCEKLTGFLIKEGFKDVAQLEGGIVTYGKDPEVKGRLFDGKCYVFDERISVPIN
HTEEDIVVGKCHHCGKPADTFINCANDFCHLQHISCPECEEKHSGCCSDKCEELVLSAR
>lcl|NZ_CP026520.1_prot_WP_042230997.1_1815 [locus_tag=PC41400_RS09335] [protein=catalase] [protein_id=WP_042230997.1] [location=complement(2049349..2050977)] [gbkey=CDS]
MKDEINNGTAVTGVGGPADQRLAEGEQETTLTNRQGHPITDNQNIRTIGNRGPSTLENYHFLEKISHFDREKIPERVVHA
RGAGAHGYFEAYGKVGEESVAKYTRAKLFQEAGKKTPVFVRFSTVVHGVHSPETLRDPRGFAVKFYTEDGNWDLVGNNLK
IFFIRDPLKFPDMVHAFRPDPVSGVQNMERMFDFLSNSPEATHMVTFLFSPWGIPANYREMQGSGVNTYKWVNANGEGVL
IKYHWEPIKTGIRNLTQNEANDIQGKNYSHATQDLYEAIERGDYPEWELSVQILSDDEHPELDFDPLDPTKLWDHEKFPF
LPVGKMVLNKNPENYFAEVEQAAFGTGVLVDGLDFSDDKLLQGRTFSYSDTQRHRVGTNYLQLPVNAPKKHVATNQRDGQ
MTYFVDRAPGQNPHVNYEPSSLGGLQEAPPRGREHEPAYNAKLVREKIDRPNDFGQAGDTYRKFEPWEKDELISNLVNNL
KICKPVIRDKMVEYFTQADAEYGRRVAEELGKAVASSEHMGTVQNREGAGKAEQAGRKTDGY
>lcl|NZ_CP026520.1_prot_WP_042230996.1_1816 [locus_tag=PC41400_RS09340] [protein=EAL domain-containing protein] [protein_id=WP_042230996.1] [location=2051586..2054273] [gbkey=CDS]
MKTIQSLLQNKARLTLGYLLCIALALLFEASELHLIYGIAFSFSSIFLLFAVSLFGVRLGLLAGVITQLFSFFFLHHPAF
EWIGFLEIAGVGLLNRKKPGRLLVWDSIYWLLIGLPLMFAIYYGFYKFVSDELILMLLFALSNGLFNALIVDILLKYLPL
FHWFGLVKERERTYSYSKVLFHLSIMAVALPSLLYIVMNSWYSEKLSIQNSKQLAVNTANSINQELNQWGAEDILGIRLK
GRIQVGYLQELVNKYTSEQLFRIVITDRSNNVLAADRQSYKLNQIYDWKSGNNVVGLDDNFYQVLPDRETAALPAQQWRE
GSYIYENAVARMPLHVYIDIPIRHYQVSVFAQYLSQLWYLLLFAAAAAATALYLSRVLVHGLSKLATSTTDLPEKLKQMK
EIEWPSSRVSEITSLTYNFRQMSLNLMSMFHQAQDMNQRLQEQTDKLRLSEEKLHQLAFYDILTGLPNRLYFTTYLQSLL
LEAVNAPKQVAVMFADLNRFKQINDTLGHDIGDLLLKEIGERFARSVEESCKLFRIGGDEFVIVIEDASDAKAAAVARSM
FAALEEPIRLMESPLYSSVSIGISMFPRDGETMETIVKNADMAMYFAKSVGCSNSYQFYEEELKIRFSEEMMIDNGIQEA
LSNHQFTLHYQPKVDTVTGAISGIEALVRWEHPELGRISPDKFIPLAERSGIIYKIDEWVLKEACRQNKAWQDEGLPKVP
VAVNISALHFSQTNLLDMIHGALEESGLEESYLTLEITEGVFIENVELVIGIISRIRERGIHISIDDFGTGYSSLNQLQR
LPISHVKLDRSFIRDIGGEPKKAAVAKAIIELAHSMNMTVVAEGVETAEESEYFTRCRCDELQGYYFSRPLPAPEFSRLL
QRGNYAGLFAGGRTD
>lcl|NZ_CP026520.1_prot_WP_042230995.1_1817 [locus_tag=PC41400_RS09345] [protein=extracellular solute-binding protein] [protein_id=WP_042230995.1] [location=2054298..2055626] [gbkey=CDS]
MVKLTKIFVLLLLLIVSTAGCFRSEPLIPASSPPAEKPAPSGDKRAVSNYLSIWTHYGGIETMAAKFKESYPDLSIDIKV
IPYSEYTGAYLQALADGKPPDVMMVDSADFGNFSAIEGLEDLLQPPYQAGKYRNDFSPSIWESAMSADGNELIAFPLSTT
PTVTYYREDIMRQYGFPSEPEELAKYMEDPANWLHMAKELREHNIYLTQWVNETVQLFERSEGIFDRKLNFMRYNEAFHK
AVEIAKKVNDFGLEAHLDVWGDMGRKAVQDGTIAMMYFGTWGVEQIKEWAPETAGKWRETRLPFGLYGWQNSTNFLLPSD
GANKEMGWKFIEFAVTEYSKEGLSGAVPAYLPARGNAKELAATNQFLGGQKTYALHEDLMTRVREFPTTPLDVKAQEIWE
SEIASGIEKRLATKQIIADARDKIEQSLAKEKQILLNTMKKK
>lcl|NZ_CP026520.1_prot_WP_042230993.1_1818 [locus_tag=PC41400_RS09350] [protein=glycosyl hydrolase family 18 protein] [protein_id=WP_042230993.1] [location=complement(2055710..2059693)] [gbkey=CDS]
MNLKSRQTAKIRRSHSPARVALRILLIVTLIVSLFPFLGLSAQKASAAESYKIVGYYPSWAAYGRNYNVADVDASKVTHI
NYAFADICWNGKHGNPDPTGPNPQTWACQDAAGANINVPNGTIVLGDPWIDTQKSFPGDTWDEPLRGNLKQLIKLKQANP
NLKTIISVGGWSWSNRFSDVAASAATREVFANSAVDFIRKYQFDGVDLDWEYPVGGGLAGNSARPEDKQNYTLLLQKIRE
KLDAAGTADNKKYFLTIASGAGPTYAANTELGNMAKYLDWINIMTYDFSGGWQTISAHNAPLYTDPAAIAAGVPNADTFN
VEKGVQGHLDAGVPASKIVLGLAFYGRGWSGCKNTANGEYQQCAGLATTGTWEKGSYDFYDLEANYINKNGYTRYWNNTA
KVPFLYNATNGTFISYDDAESIGHKTSFIKSKGLGGGMLWEFSGDRNKTLLKKVTTDLAGSAPVGDTIAPSVPANLTATA
KTSTSVTLSWSASTDNVGVTGYTVTYGTKTVDVSATTATITGLTPSTAYSFTVKAKDAAGNVSAASSAVSVTTDANSGGG
TDTTAPTVPANLAVTAKTSTSVTLGWTASTDNVGVTGYTVSYGSTSVNVTGTTATISGLTAGTAYTFSVKAKDAAGNLSA
AATVSATTDSGSGGAGCPAAWDTSKSYTGGQRVSYNGKIYEAKWWTQGERPDSSNQYGVWKLISDCGAGAGSPASVSSGG
AALASNLIAGAGAALPAIAPGSSAASTSSALAPSTKVLVGYWHNFDNGSGFIRLRDISPKFDVINVSFAEPVGGSTTGTI
GFTPYNYTDADFKADVAYLQSQGKKVIISIGGANGQVQLTTAGARNNFVSSMKSIISKYGFDGLDIDFEGHSLYLNPGDA
DFKNPTTPVITNLISAIRELHDSFGDKFMLTMAPETFFVQLGYAFYGGTCSGCDTRAGAYLPVIYGVRDILDWLQVQLYN
SGPITGLDDQYHTMGVADFHVAMTDMLLTGFPIAKNPNSFFPALRPDQVVVGLPANGNAGGGFTSVSEVHKALDYLMKGK
AFGSYTLRGSVQPGLRGLMTWSINWDKFNNFEFSNSHRAYLDSFGPGTPVDTQAPSVPGNLRSSNVTATGVTLTWNASTD
NVGVAGYEVYQGSVKIADTASTTYNVGNLTANTSYTFTVKAKDAAGNVSAASTPLAVTTANGGGSGDTTAPTAPTALKVT
AKSSSSVSLGWTASTDNVGVTGYTVSYGANTVNVTGTTAVITGLTANTAYTFTVKALDAAGNVSAAASIQATTDPASPGG
PAAWAAGVSYKLNDEVTYNGSTYICRQPHTSLQGWEPSNVPALWTKK
>lcl|NZ_CP026520.1_prot_WP_042230992.1_1819 [locus_tag=PC41400_RS09355] [protein=methyl-accepting chemotaxis protein] [protein_id=WP_042230992.1] [location=complement(2060212..2061939)] [gbkey=CDS]
MGKFFYLSTMKRKLIASFGVLILLFVAMAFFLVHQISKISSQIALQNERVAMKTTALELKETVQELSIIASGLELSKDPS
YIEGYEQTRKVYSRLLDVIKETARTEEEKAWRGELIVASADYINNFDAAAKLVQEKKVSPKEMELSMLHYYGEGQILRET
IFKLVDKFYIAFSADAQAAVEDSRKVIAQTVTVTVAAAVLVLLASAVVAVLLIRSFLIPVRKLQEAVARMASGDLRHEIG
SRTQDELGELSRSFDRMTAQVRGMLEHTSGIATALAGQSVTFRRVSTDTAAANADIVQAMSAIAHGAFEQAADSEHASTS
VAGLESETREIAEYAAVMSQASREAAEVTVQGSRSVHSLEEASGATRRVLGQVQEAMATLAGDSRRIGAILDAITEIASR
THLLALNAAIEAAAAGSSGRGFAVIAAEVRQLAQQSSDEARRAAGTIASVESRMLEMQARVTDARRHADAQELTVRESLG
SFEAIERAMEVVTNQIQFVNGRIERSLHSSAQLVGTIRHMAEIAEQTSAGVQEVTSASTENDASIRSIAVQADEMNGLAE
SLYREINKFKIGEAS
>lcl|NZ_CP026520.1_prot_WP_042230991.1_1820 [locus_tag=PC41400_RS09360] [protein=phosphate ABC transporter substrate-binding protein] [protein_id=WP_042230991.1] [location=2062215..2063111] [gbkey=CDS]
MFKFMSKKGLSISLAAVLTLGVLAGCGKTDTNASSPATSNKPAETTAAQEISGTVTASGSSALLPLVKEAAKLFMEKNPK
VTINPTAGGSGKGLKDVADGISNIGNSDVAAGDEYKDKNLKEHVVAIAPFALIVNPDVNIEGLTKQQAADIFMGKIKNWK
EVGGKDAPINVVHRPDSSGSRKLVQQIVLEGKDFTKEGVVQESTGAVKTAVGSQSGSIGYIDTPYLDNTVKALKFDNVAF
SKETIKNGSYKLFGTERMYTKGEPTGATKAFLDYIMSPDFQSKKVEELKFLPADLLKK
>lcl|NZ_CP026520.1_prot_WP_042230990.1_1821 [gene=pstC] [locus_tag=PC41400_RS09365] [protein=phosphate ABC transporter permease subunit PstC] [protein_id=WP_042230990.1] [location=2063305..2064255] [gbkey=CDS]
MNTYAQQTTKTKEQSRQNRQSQKGPTRWMKLQKRQDSVMKLLFVGSASLVSIVIFSIILFVGVQGMRTFSGVSPIEFFFS
TDWTPSADKFGAFPFLFSTFLLTMLAIVLAVPLAVAGAVFMAKIAPKWVREIMRPATDLFVGIPSVVYGLIGLTVIVPFI
GKVVNGTGYGILPAGIILAIMILPTILSISEDAIRALPPRLEEASLALGATRWQTIWRVLLPAARPGILTGVVLGIGRAI
GETMAVFMVIGNSPQMPKSLIDSTTVLTTAIVKDMGNTNYGSTWNNALYLMAFVLLITSLLLILIIRVVSRRSEVK
>lcl|NZ_CP026520.1_prot_WP_042230989.1_1822 [gene=pstA] [locus_tag=PC41400_RS09370] [protein=phosphate ABC transporter permease PstA] [protein_id=WP_042230989.1] [location=2064252..2065109] [gbkey=CDS]
MNSKTTDRIATLFFWATGIFILLILAWFLIRILGAGLPHLTWDFITGKPEDIVAGGGVGPQLFNSFYILFLSLLFSLPIG
LGAGIYLAVYAKKSRFTEIVRMSVETLSSVPSIVFGLFGFLLFVDLMNFKISILSGSLALALLNLPVLVRVTEESIRTVP
ESYWEASLALGSTKWQAIRKVLIPTALPSLITGITLVAGRALGESAILIYTSGLSVSRHFPDFNPLTMGETLSTHLWYIQ
AEAIVPDAKKIAEGSAALLLVVVLIFNLLIGIPSRIIQKRLTGGK
>lcl|NZ_CP026520.1_prot_WP_042231189.1_1823 [gene=pstB] [locus_tag=PC41400_RS09375] [protein=phosphate ABC transporter ATP-binding protein PstB] [protein_id=WP_042231189.1] [location=2065127..2065894] [gbkey=CDS]
MNIATDTKIKVNGLDLFYGDNQALHGIDLDIAPNSVSALIGPSGCGKSTFLRTLNRMNDLIDSVRINGTIKVDGQDIYSP
ESDIVSLRKRVGMVFQRPNPFPMSIYDNIAYGPRIHGQKKRAVLDEIVERSLRQAALWDEVKDRLNKSALGLSGGQQQRL
CIARLLAVEPEVLLMDEPTSALDPISTLKVEELTQTLKERYTIIIVTHNMQQAARISDRTSFFLNGILVETDQTDKIFTN
PNDQRTEDYITGRFG
>lcl|NZ_CP026520.1_prot_WP_042230988.1_1824 [gene=phoU] [locus_tag=PC41400_RS09380] [protein=phosphate signaling complex protein PhoU] [protein_id=WP_042230988.1] [location=2065911..2066570] [gbkey=CDS]
MNNRLNFQHSIDDLQNELLDLGSMVENSIHQAVEALTRLDADLARRTVEDDDAVDELTLRIEEMCLKLIALQQPMAGDLR
IIGTALSIATDLERIADHAVDIAKITIRLSGETLAKELIDIPQMAEISKEMLRESLVAYMNRNVHMAASLAEKDDKVDKI
FSRIMTDVIGMMNADFTRNRQLTHLIMVAHYLERVADHVTNIGESVIYMVTGKRKDLNV
>lcl|NZ_CP026520.1_prot_WP_042230987.1_1825 [locus_tag=PC41400_RS09385] [protein=HAD family hydrolase] [protein_id=WP_042230987.1] [location=complement(2066943..2067692)] [gbkey=CDS]
MSPFFFPAPKKMIFFDMNNTLVDRRQCFDSAFTEVLGDYTGRWEQEDGAASPENALQIYKQEWQKLRKLRPRGHRPSEEM
RIACLEKALDAFPISVNETFAKGFFEQIEETEGNYVRLFPGVEETLAALGERYKLAVISNGSRKRLERDIGRLRLSSFFP
EKLLFSSTKNGPRKPHPGLYRTALKETGIAPGQGVMVGNSWKNDIIGSTRCGLDAVWIYPPHVKKISQRKVGRQKVIIVR
SVKQLVHVL
>lcl|NZ_CP026520.1_prot_WP_042230986.1_1826 [gene=polA] [locus_tag=PC41400_RS09390] [protein=DNA polymerase I] [protein_id=WP_042230986.1] [location=2067900..2070566] [gbkey=CDS]
MEKLIVIDGNSIASRAFYALPLLSNSAGLHTNAVYGFTTMLLKLIEEEKPTHFLVAFDAGKVTFRHKDYTEYKGGRAKTP
PELSEQFPLIKELMSAFGIRQFELEGYEADDIIGTLTKSAEQEGKDVLVVTGDKDMLQLVSDRVTVAVTRKGISEVDRYD
PAEIQAKYGLVPAQIIDLKGLMGDSSDNIPGIPGVGEKTALKLLHEYGSVESVLEHADEMKGKMKEKVRDHADSARMSKE
IATIYREVPMDTDWSGLAFSGFETPELAAMFRKLEFKSLLERLDFSGAGGAASSEGAESGPAEELVVRLVDADNITDLIG
ALQDVSALHVEAVGDNAHTAELVGAAFHTLEAGYYVPAAVLTAPGAEALRAWLADKGAPKSTWDRHRAELVLAWNGITLA
GVTFDVMLAAYLLDPTESSLTLSGIADKYGALHVAPDEAVFGKGAKFRLPEPEAVAEHLARKADRISRLDVSLREKLEQA
DMKQLYYELELPLSGVLADMELRGIKVDTAGLKQLGDELAVQIERTMTEIYRHAGMEFNINSPKQLGEVLFEKLQLPVQK
KTKTGYSTDAEVLEKLAPYHEIVQGILHYRQLAKLQSTYVEGLLKEVRKETGKVHTYYRQTIAATGRLSSQYPNLQNIPI
RLEEGRKIRKVFVPSEPGWSILAADYSQIELRVLAHISGDENLKEAFTNNMDIHTKTAMDVFGVAESEVDANMRRSAKAV
NFGIVYGISDYGLSQNLAISRKEASQFIEQYFAVFQGVRQYMDDIVKLAREQGYVTTLLHRRRYLPEITASNFNLRSFAE
RTAMNTPIQGTAADIIKLAMVQMDERLKAEGLKSRMLLQVHDELIFEVPPDELETMRRIVPEVMEGALALDVPLLAEVDY
GSNWYEAK
>lcl|NZ_CP026520.1_prot_WP_042230985.1_1827 [gene=mutM] [locus_tag=PC41400_RS09395] [protein=DNA-formamidopyrimidine glycosylase] [protein_id=WP_042230985.1] [location=2070607..2071467] [gbkey=CDS]
MPELPEVETVRRTLSRLVVGKTIREVEVRLKRIIQKPEEPELFADLLEGQTIQGVSRRGKFLRFLLDDGVLVSHLRMEGR
YGLYEGSDEVEPHTHVIFRFTDGTELRYRDVRQFGTMHLFAGEEDLSSKPLHKLGIEPLDESFTFDAFRQAIGKRSTKIK
PLLLNQAYIVGLGNIYVDEALFLAGIHPEREAASLTRLELQKLHEAIVHTLKSSVEAGGSSIKSYVNGQGESGNFQHSLN
IYGRNGKPCVTCGSLIEKTVVGGRGTHFCLKCQPLKRKRSTPKNKL
>lcl|NZ_CP026520.1_prot_WP_042230984.1_1828 [locus_tag=PC41400_RS09400] [protein=MntP/YtaF family protein] [protein_id=WP_042230984.1] [location=2071531..2072301] [gbkey=CDS]
MLPFFSLLILALAVSLDGFGVGMMYGLRKIRIPLPSLAIISLCSGIIIYISMSFGVWLSRYLSPHYAGMLGAVILMGIGI
WALVQFLAQKQGNDGDGEGKERTQAENGKAAVSAGTSRLEAAFQSNPVAARRPEAPRAEPEREPRMFSIEIKRLGLVIQI
LRTPSAADVDRSGNISSSEATLLGIALSLDAFGAGIGAALIGYTPLLTSAVIAVTSGLFIFLGLKIGYQYSGLGWIRKVS
MLPGVVLILMGISKLF
>lcl|NZ_CP026520.1_prot_WP_042230983.1_1829 [gene=coaE] [locus_tag=PC41400_RS09405] [protein=dephospho-CoA kinase] [protein_id=WP_042230983.1] [location=2072346..2072939] [gbkey=CDS]
MNIGLTGGIACGKSTVSAMLVRRGAILIDADRIAREVVEPGSPVLAQVAAHFGQDMLLPDGSLHRKKLGKTVFGNEEARK
ALEDLLHPSIRALMKERMAAAEREFPDKLVVVDVPLLYESGLQSMFSSVMVVYVPREVQLRRLMERDGLNAEQAESRLAA
QWPIERKKELADSCIDNSGTLEQTEQFVERFWLQRGL
>lcl|NZ_CP026520.1_prot_WP_042230982.1_1830 [locus_tag=PC41400_RS09410] [protein=lytic transglycosylase domain-containing protein] [protein_id=WP_042230982.1] [location=2072939..2073517] [gbkey=CDS]
MKLLRKKRVFALLLLGFVFLLFSNGLFIGQIIYPIHYREEIRQNAAQYQVDPFLIAAIIKVETNFKHDRESKKGAVGIMQ
LMPDTAEWIRETSNTKSADFRDPKHNIQLGSWYLAWLNKQYKGNWLYSIAAYNAGQGNVSKWKNNGVWDGTEETIKKIPF
GETRHYVQRVLYYQQKFKKLYGEEWNQSAGSK
>lcl|NZ_CP026520.1_prot_WP_009671472.1_1831 [locus_tag=PC41400_RS09415] [protein=alpha/beta-type small acid-soluble spore protein] [protein_id=WP_009671472.1] [location=complement(2073588..2073818)] [gbkey=CDS]
MGAGQSRSSNVLVVPQANQALDQLKYEVAQELGIAIPQDGYYGNMATRDTGAIGGHITRRLVQIAEQQLAGQGQSR
>lcl|NZ_CP026520.1_prot_WP_042230981.1_1832 [gene=nrdR] [locus_tag=PC41400_RS09420] [protein=transcriptional regulator NrdR] [protein_id=WP_042230981.1] [location=2073930..2074412] [gbkey=CDS]
MKCPFCEHNGTKVLDSRSANDNKSIRRRRECEKCQRRFTTFEMVEETPLIVIKKDGSREEFNRDKILRGLIRACEKRPVS
VDRLEQIVSEVEMQVRNMAHAEVDSMRIGELVMEQLYPVDEVAYIRFASVYRQFKDINMFLKELEQILGKTNLSALKSDG
>lcl|NZ_CP026520.1_prot_WP_042230980.1_1833 [locus_tag=PC41400_RS09425] [protein=YuzF family protein] [protein_id=WP_042230980.1] [location=2074601..2075212] [gbkey=CDS]
MSMTQGNGQGVPWGFGKPEVPDHTKMTQGQAQLPGYPLQQGSPFGGGGNLPPFTQGQPPGGMTGAFPPQGAYPESFGSAD
GWPGQAGFPPGSQPGGGGFAPFPQGGFGSIGGGAGQGGYGYGALEAGGPSGGAPVAVPQPFFSIDPYVYLTLKMIVGRCV
VVETVRGSNRGVLTDVKPDHVVVQEDGKKFFIRTAQIIWVMPE
>lcl|NZ_CP026520.1_prot_WP_042230979.1_1834 [locus_tag=PC41400_RS09430] [protein=FMN-dependent NADH-azoreductase] [protein_id=WP_042230979.1] [location=2075402..2076049] [gbkey=CDS]
MNKLLYITANPREVDRSFGLQVGEQFLEAVSKENPDLQVERVDVYEEHIPLIDADVLNGWDKLRAGEALNETEAAAINRM
NEVLEQFLSADGYVFVTPMWNLSYPPMLKAYIDNIAIAGKTFRYTANGPEGLMSGKKVVHIQARGGIYSEGPAADFEFTD
KYLRGIMAFIGITDYEHVYVEGIAAFPDKAQELVQGAKTRAAAVAAQFAKTPSRV
>lcl|NZ_CP026520.1_prot_WP_042230978.1_1835 [locus_tag=PC41400_RS09435] [protein=DUF441 domain-containing protein] [protein_id=WP_042230978.1] [location=complement(2076216..2076668)] [gbkey=CDS]
MAQVDMPSILLLGLALLGLLSSNSTVTIAMLVLLLLRVMNVHQAFPFLEKYGLTIGVVILTIGVMTPIASGKMPVQTILQ
SFMHWKSLLAVLIGISVAYLGGRGALLMTNQPNVVAGLLIGTVLGVALFRGVPVGPLIAAGLLSLLIGKG
>lcl|NZ_CP026520.1_prot_WP_042230977.1_1836 [locus_tag=PC41400_RS09440] [protein=ribosomal protein L7/L12] [protein_id=WP_042230977.1] [location=2076854..2077189] [gbkey=CDS]
MEAIEMIAVGALILALLLLVKVFSLQSRINAIQDDLEWLKGRAEGQAINRLKPSTADRSDKPASAQMSLLDERVHHLLAQ
GDKIKAIKLVRVEQGIGLKEAKDYVENLERS
>lcl|NZ_CP026520.1_prot_WP_042230976.1_1837 [locus_tag=PC41400_RS09445] [protein=SDR family oxidoreductase] [protein_id=WP_042230976.1] [location=complement(2077191..2077955)] [gbkey=CDS]
MSKLQGKVALVTGASRGIGRAIALRLAQEGALVAVHYGKNRNAALEVVREIQSREGSAFAIGADLTSAGGIDQLFAALDT
GLLEFTGGNHLDILVNNAGIAQTLTLEETTAESWDEVMDIHVKAPAFIIQKALPRLKENGRIINISSAVTRIAFPNLLGY
SVSKGALNTLTYVLAQHLGSRNITVNAIQPGIIDTDMNAETLQSTGGQEFAAGLSAFGRWGQPEDVADIAAFLASPDSRW
VTGQLIDASGGSRL
>lcl|NZ_CP026520.1_prot_WP_042230975.1_1838 [locus_tag=PC41400_RS09450] [protein=elongation factor G-binding protein] [protein_id=WP_042230975.1] [location=complement(2078193..2078852)] [gbkey=CDS]
MIQPFIRNHQYNVIKKQSLLLQKTHETVSDPKVVESVKSSAPFKLLEAFPEADGLQKQLLEKISGLRTAEEFKHYPESLE
SYLAPFPALTAKQLAKLFPKNKKLKIPDLSEIDYRYVTYLGWTDIATNKLFLVYQLNGKLIGIEGRYTPANKGVCFLCNK
HEEVALFTAVTKSRPAKASSDYYKAIGNYLCVDSTVCNQNITNVTALDKFVREVVGQES
>lcl|NZ_CP026520.1_prot_WP_042230974.1_1839 [locus_tag=PC41400_RS09455] [protein=winged helix-turn-helix transcriptional regulator] [protein_id=WP_042230974.1] [location=complement(2079179..2079613)] [gbkey=CDS]
MDQIDSKILSLMHENARIPISEMSRMIAMSQPAVTERLRKLEEQGVITGYRAKLSPRKLGKQTTAFVLFKTTDCKGFAAF
SEKSPEIVDLYRISGEYNFLMKVLSKTTESLTGFLDECSAFGFSTTLIVLSTAFEDKSLVTSEL
>lcl|NZ_CP026520.1_prot_WP_042230973.1_1840 [locus_tag=PC41400_RS09460] [protein=tRNA-dihydrouridine synthase] [protein_id=WP_042230973.1] [location=2079761..2080753] [gbkey=CDS]
MTRTLNDPITIGGIPLKNRLIMAPLQQYKGTPEAFAAAHHIEHYGRRAKHVGLIVLESTAVSSNGRLWANDIGIYTDRHV
EALRKVTDAVHAHNTPVFIQLSHGGRKSSPEVTTTLVAPSAIAFDEQYGMPQELSAEGIHNIIQEYRMAAKRSKEAGFDG
IELHAAHGFLIHQFLSPLSNKRTDAYGGSSENRARFLKEVLAAVRGEVGRNYPVIVRISATDYSEGGLIPEEWARMLKPL
ESELDAIHVSTGGIIPVQPSDVYTAYQLPHAAAIKQHFKIPVIAVGQIYTRSLADRILEDGLADVIAIGRPLLEDPDYAE
GMLTVREVSV
>lcl|NZ_CP026520.1_prot_PC41400_RS09465_1841 [locus_tag=PC41400_RS09465] [protein=Rpn family recombination-promoting nuclease/putative transposase] [pseudo=true] [partial=5'] [location=<2080801..2081628] [gbkey=CDS]
DENKDVLLTFLNRTFAESGEPPLTEIVLLNPYTEKDAPLDKQSIFDIWAKTAEGKLINIEMQLFNKYDIEKRTLYYWSKR
YSGQLQEGQTYKELKKCVTINILNYSFISNDRYHNVYHLREDHTGLELCDDIEVHFMELSKLDEKAAPVEGGLINWLLFL
KGVDKTNWEVLKMNEPTLKKAMDTLEFLSQDREARRLYEERQKYLHDEASMIEWATEKGMAKGMAQGMAKGIAEGEQKKA
IQIAKNMLALGLQVSVIAKASGLTEAEVESLKTVQ
>lcl|NZ_CP026520.1_prot_WP_042231187.1_1842 [locus_tag=PC41400_RS09470] [protein=SDR family NAD(P)-dependent oxidoreductase] [protein_id=WP_042231187.1] [location=complement(2081760..2082479)] [gbkey=CDS]
MRSYLIFGASKGLGDAFVKGLPEHGDQVWIVSRSRPGSLDLNDGIKRIWITADLAQPDAAQVISKGIQTEKIDVLVYNVG
IWEKEGFEAQYTFDKDEASDIANIINVNITSTITCIQAALPNLRKSEAGKIILIGSTAGLDHTNNNQVSFVASKFGLRGI
THALREHVRKDGIAVTCVNPGELAAEIPYEDGADKAIAEYSGTRIPVQDIVSIVKCVVNLSRVSCVKEIHIPAMTDLNA
>lcl|NZ_CP026520.1_prot_WP_156972824.1_1843 [locus_tag=PC41400_RS29725] [protein=hypothetical protein] [protein_id=WP_156972824.1] [location=complement(2082821..2082967)] [gbkey=CDS]
MFTKITIQEDERGLLFRDGRYVKLLQPGTNRSLPFSNGCGDYVCVEAF
>lcl|NZ_CP026520.1_prot_WP_042230972.1_1844 [locus_tag=PC41400_RS09475] [protein=SWIM zinc finger domain-containing protein] [protein_id=WP_042230972.1] [location=2083480..2084913] [gbkey=CDS]
MINLSEAYVDSLAHNSAAMKNGKDLFKKKSFTKLCKTEDGKLLFGECKGSGKEPYRCSVDWMTEGQPVFRCSCPSRQFPC
KHLLGLMYAYAAETPFETAPIPEDIMGKREKVEKREQKKSEATSKEKPTKRKTSKSALLKKIGAQLEGIAIFEKLLHQIV
QSGLGSLDKQMLKMIEDQAKQLGNYYIPGIQTAMRNLIVLLRSDEEQETIYAKASAYLSTLHTLLKKSRGYLQQKLDNPE
QAPELDSTIEEWIGHAWQLAELRECGRVQSDRDLMQLSFNSYADEARGEYVDESFWADVESGTVYTTRTYRPFRAAKFIK
EEDSYFAIVNAKELMIYPGELNCRIRWEGAVPRPVTEQDFQAVKSKALSSVTEAVKIVKNQIKNPLSDKRPVLLLAYDQL
MHSSNGPVLLDGQGKELQLSDLESVEQSTAHMLPMLHQDVLRQQAAVVRFEHDMDNNRLHAQLLSIVTDQGIIRLLY
>lcl|NZ_CP026520.1_prot_WP_042230971.1_1845 [locus_tag=PC41400_RS09480] [protein=HEAT repeat domain-containing protein] [protein_id=WP_042230971.1] [location=2084933..2086798] [gbkey=CDS]
MSIDLLFDLNREVRRLFIAGSGMAAGDFNLKKILPQLHKFGESAAVFKRIAQAVEEVLESNREDGAVKLLDLGTLLTSVL
HTQGKTEPQTERHAIAGTQIQIGTNIPYRKLQPLIEALNTKGSGRWEVIHQAHEEGLFNDFRVMFSAVDALDDGYSEIAD
FIHNQVLLAYGPEILPVLHQQFRQQGGKGDARRLEHIYRLQQEEGLDLYLAAAGEGSVEVRAAVIKILGHFADQESFILE
QADDKKKEIREAAYHALSTLGTEQALDRLFAVLKSKDREMAVEPIRRSSSFRLANQVIAHADAAFERILVSLAQKKTKVT
ENEADKEKKKEKNKELEKAVEELYVDLQSLDGFESEEVYTLLTKLLSTPGFVSQHTYWLQNMAVNMLQNMQYSEALPFVL
DLQDKHKKAFICNSLWAAFKLLSPADVYERFAPEFEERSRKTVKELLDYIPRIIPNMVRQLNADLEEEQQWTQNWDARWI
DLFMKMNQDDLVCRLVYGTATRPSSTVITYLIHKCKNQLDLSNDNTAQMLITLLKLGHEEAPDLVMEALENGGIKRYYYL
YSNSLLFHLFHMLPYTYADQLEHFAQSPKGERFKEQLLEIVENLRAKQAEGTVEAQKGMSV
>lcl|NZ_CP026520.1_prot_WP_042230970.1_1846 [locus_tag=PC41400_RS09485] [protein=AAA family ATPase] [protein_id=WP_042230970.1] [location=2086795..2087892] [gbkey=CDS]
MSSNQEQNVVRLPAERLYAAEIEALKSVDKDSKPAGWQLSPKAVLTFITGGKAGDVTITPKYIGNRRLVEMAIATLLTDR
ALLLIGEPGTAKSWLSENLTAAIHGDSSKVIQGTAGTSEEQIRYSWNYAMLLAQGPSEQAIIKSPIYRAMETGGIARFEE
ISRCASEVQDALISILSEKTISVPELGSEVSAARGFSVIATANTRDRGVNDMSAALKRRFNIIVLPAPSDLDTEVDIVCK
RVAEISASYELKASLPDDEAVRKVVTIFRELRSGMTLDGKEKVKPTSGVISTAEAISLLTGSMALAGSFGSGTISDEDIA
AGLQGAIVKDEEKDKLVWKEYLENVMKKRGSTWRSLYNVCSELNG
>lcl|NZ_CP026520.1_prot_WP_042230969.1_1847 [locus_tag=PC41400_RS09490] [protein=DUF5682 family protein] [protein_id=WP_042230969.1] [location=2087850..2090237] [gbkey=CDS]
MAEPVQRLLGTERVSEKKRVNVFGIRHLSPAGAYHLRALLEEKQPTAILVEGPSDANDYIHQLTMSGVKPPVALLAYTEE
LPVRTLIYPFASYSPEYEAFRWAKQHGIHAEFMDLPSTISLTLYENRKGAGRKAKAEAVQAGEGKNETSDRASYVQLQNK
MYEQITELSGEPDYEAYWERYFEHCTSKEAYREAIDHYSIQMRALMEEAEQSLAPEEAAYNEIREAYMRRKIEETIRSGH
APEKIVVVTGAHHASALRPELLPMTDRELEGLPKVRTKITLMPYSYYKLSTYSGYGAGNQAPAYYELFWTCLQQNELWKL
PTLYLSKVAAYLRDQGTYRSTASVIEGVRLAQALAVMHDGTSPTWRELRDAAIVLMGYGELSVISEALAKTDVGTAIGSL
PDGVSQTPIQDDLNRELKRFKLEKYKTTVATTLELDLRENRRVKTVEAAFIDLNRSVLLHRLELLGISFVKKQHVSQASA
SWAENWVLQWSPEAEIEAVESTLKGETIELAAAFVIHEKLQACADIAEASRIIRLACTSDLPSLMEQARGVLQELAVDSG
NFEQIASAALELSVLIGYGSIRRIATDSLIPLLEQLFLRGSLLLVDAANCNDEAAKTMASAIQTFHSIAQEHYEIVDDQI
WLQKLRELSMRDDRNAKLSGLAFSILLERNEVDEARSAQEMSRRLSPGIPADLGAGWFEGLAMRNRYALLSRSFLWEQLD
AYIQSLENEQFYRSLVFLRRAFGTFEAREKANIAELMGELWGLGAIATGEALQRPLGEEEKEKLDELNDFDFGDL
>lcl|NZ_CP026520.1_prot_WP_042230968.1_1848 [locus_tag=PC41400_RS09495] [protein=VWA domain-containing protein] [protein_id=WP_042230968.1] [location=2090241..2091452] [gbkey=CDS]
MTADNLEKMKRWRLILGAATEEKLKSFDSTLAQLLDEEARIMDEALAAIYDDTQLIDDSSRSGQKGQTRAAGLGSSSPKI
SKWLGDIRTFFPPDVVSVIQADAIERKGLTQLLFEPELLSQVKPDVQMVATLMSLKGRIPEQTKETARQLVKAVVDEIMK
RMSEDLRRAVTGALNRRKHSPIPSVTGLDWNTTIRKNLKHYDTNRKLLIPEKVYFFDRARKSKEWTVILDIDQSGSMANS
VIYASIVGSIFASMPSLETKVVAFDTEVVDLSEQCANDPVDMLFGIQLGGGTDINKSVAYCEQFIKEPQKTLFILVSDLY
EFGNRSEFLRRMDELHQAGVKVLCLLALSDEGVPSYDEAIAKRLASYGIPSFGCSPDRLPELIEGALKGMDLVQLSARVG
EKK
>lcl|NZ_CP026520.1_prot_WP_042230966.1_1849 [locus_tag=PC41400_RS09500] [protein=helix-turn-helix domain-containing protein] [protein_id=WP_042230966.1] [location=complement(2092445..2092876)] [gbkey=CDS]
MTYASKTKQSDISLSNCGYSKVLDIISNKWTVLVIYAMENGKIRYGEMLRRVEGISKKMLTQTVRKLERDGLVQRHISPT
VPPSVEYSLTPLGETLLQPMKELRQWGRVNYTQVVEARANYDFAYTTGSLKKSSADEVMELQV
>lcl|NZ_CP026520.1_prot_WP_042231186.1_1850 [locus_tag=PC41400_RS09505] [protein=N-acyl homoserine lactonase family protein] [protein_id=WP_042231186.1] [location=2093106..2093954] [gbkey=CDS]
MTNIIKTRPKLFVLDNGRMSMDKNWIIAMHNPATINNPNAPTQFVEFPIYTVLIDHPEGKILFDTACNPNSMGIGGRWAQ
STQLSSPWVASEECYLHNRLEQLNVRPEEIKYVVASHLHLDHAGCLEMFTNATIIVHEDEFNGALQTYARNEKEGAYVWA
DIDAWIKNQLQWKTVKPGEDNLDLVDGIKILNFGSGHAWGMLGLQIEMPETGGIILASDAIYTAESFGPPVKPPGIIYDL
IGYRNAVEKIRTLATRTNSQVWFGHDADQFRHLRKSTEGYYE
>lcl|NZ_CP026520.1_prot_WP_241688887.1_1851 [locus_tag=PC41400_RS09515] [protein=hypothetical protein] [protein_id=WP_241688887.1] [location=2094707..2095054] [gbkey=CDS]
MVLLSNNTLYWVFTVLIFGITLIMGIVMMRRTRKLWLSFGISTLFNTVMTVLVCLWWARITDDAYSVLFHNAFFLLGYFN
VVIIDFFALVSIQRKTLGEGVKKRKKTYEEEYGDT
>lcl|NZ_CP026520.1_prot_WP_042230964.1_1852 [locus_tag=PC41400_RS09520] [protein=VanZ family protein] [protein_id=WP_042230964.1] [location=2095119..2095622] [gbkey=CDS]
MKPRLRTYGWWLAATLVWMIFIFAKSAETYQEQDMRPFIRDLIQEDVLRQWIPQVEFTYDGGLVTYKKPYDFTEFIIRKA
GHVSEFALLAFLWIRTLLAMKRPFWTAWLAGGFLAVLYAASDEWHQSFVPGRTGHAIDVGMDSVGVLLVMAVFAIIRLIR
AWNVRSR
>lcl|NZ_CP026520.1_prot_WP_042230963.1_1853 [locus_tag=PC41400_RS09525] [protein=hypothetical protein] [protein_id=WP_042230963.1] [location=2095684..2096052] [gbkey=CDS]
MDTYIKHAKVTRITNHKDVKVADVEFAVEGFDKPLLAKFEKPHGASSSGSAGETSFKLGPVYRNDADLTLDWYNDNLHDA
YEDVTAELLSSSTNIYELQKDQLEARLLEHGDVKRELEEGRL
>lcl|NZ_CP026520.1_prot_WP_042230962.1_1854 [locus_tag=PC41400_RS09530] [protein=hypothetical protein] [protein_id=WP_042230962.1] [location=complement(2096252..2096545)] [gbkey=CDS]
MNEQQARKWSVMRRKGPGMYVMLNFALPVGLVLTALVSLLEYSLAGELIGIWLPIRLIVFCFIGFFLGMFRWQSVDKKYQ
QVAPKYGLPVQLEKGTK
>lcl|NZ_CP026520.1_prot_WP_042230961.1_1855 [locus_tag=PC41400_RS09535] [protein=tetratricopeptide repeat protein] [protein_id=WP_042230961.1] [location=2096751..2097416] [gbkey=CDS]
MLGKFFLFSLLLYITRSPLLALIILLVVLYVIDRRFIGLSPSLLRPLKRNRRISQLRQELRANPHHASSKLELARLLIDK
RKYREALVYLREVEPVMSESADVSYEIGLCLLKMDQLQEGLAYMTKALELNPRVKYGDPYLQMGEALAESDPRQAVDMLE
RFRDEQSSSVEAYYRLGLLYGKLGRTEKAKEAFRESVDVYKSLPKYSRRKQRRWAILARFK
>lcl|NZ_CP026520.1_prot_WP_164977116.1_1856 [locus_tag=PC41400_RS29730] [protein=hypothetical protein] [protein_id=WP_164977116.1] [location=2097551..2097712] [gbkey=CDS]
MDLMYIAAAAGDSGKSKLSMEYLMQSVLLVIIVVIMFVVMLAWSKWSKKRRRK
>lcl|NZ_CP026520.1_prot_WP_042230960.1_1857 [locus_tag=PC41400_RS09540] [protein=DUF86 domain-containing protein] [protein_id=WP_042230960.1] [location=2097718..2098149] [gbkey=CDS]
MYYVNVDQIKQRLAFVPELLTGCTELEQAFSNGDKRLITYLAQERVLHLAIETVTDVGSLIIDGFLMRDASSYEDIIDIL
GTEKVFSEETTAALLSLVKLRKPLVQEYDTFSREGVHPLLKELPVILPDFTRSVEAFIEKEMI
>lcl|NZ_CP026520.1_prot_WP_009671457.1_1858 [locus_tag=PC41400_RS09545] [protein=transcriptional regulator] [protein_id=WP_009671457.1] [location=2098260..2098898] [gbkey=CDS]
MYRDAELSTRKIVLSLLRTRGPLSVGELAKELGITEMAVRRHLNTLDRDGFIESKLARQAMGRPLHMYSLTKEADDLFPK
KYNALTLDLLEELVAEEGEGKLTRLFQRRKDKLSERYGSRMEGKSFPEKVAELARIQNDNGYMVSLEQGENGDYILNEHN
CPIAQVADQYQQACQCELALFQELLGTPVERTECLAKGGTKCVYQIRRAESF
>lcl|NZ_CP026520.1_prot_WP_042230959.1_1859 [locus_tag=PC41400_RS09550] [protein=MBL fold metallo-hydrolase] [protein_id=WP_042230959.1] [location=complement(2099213..2099998)] [gbkey=CDS]
MDVLTFFGTGDSMGVPRAYCDCGVCTEARTTGVNRRLRSSALLDTGTAEGSLLLDCGPNWGLQMERAGLRDLRHVLITHA
HYDHIGGLPEYADLCRWLGIRGKVYAAAEVVDRIRAIFPWIDRNLDYIAVDEGFSFGGWEIRLWKVTHGANGFSFALRFD
RGEEYSFVYCPDSIGLNEQEKAPMRGLDLLVLGTSFYKEEGAGRSVYDVTEALELLREVKPSRTVFTHLSHGIDLLEDYA
LSSGVRFAQAGLSIALSRDSG
>lcl|NZ_CP026520.1_prot_WP_042231185.1_1860 [locus_tag=PC41400_RS09555] [protein=bifunctional 4-hydroxy-2-oxoglutarate aldolase/2-dehydro-3-deoxy-phosphogluconate aldolase] [protein_id=WP_042231185.1] [location=complement(2099986..2100645)] [gbkey=CDS]
MEREQGLQELERTRLIAILRGVKPGQIIRTAEAMLEGGVTVLEVTLNTDGALDMIRELRSRLGGRMFIGAGTVLDLEDAR
QAIEAGASFLVTPNTDEAVIAYSVSQGIPIFPGAMTPTEIVRAWKAGAQAVKVFPSTGLGLTYIRELMGPLDRVPMVAVG
GVTAGNIGEYLQAGCYAAGVGSSLCKLADIDAGRFDVITANARRFAEAVSQWKEAAWTS
>lcl|NZ_CP026520.1_prot_WP_042230958.1_1861 [gene=selD] [locus_tag=PC41400_RS09560] [protein=selenide, water dikinase SelD] [protein_id=WP_042230958.1] [location=2100835..2101896] [gbkey=CDS]
MKDAQPVKLTSLSSKGGCGCKIGPADLSQVLRGMPAPLPNPNLLVGLDTSDDAGVYKLSDDLALVQTVDFFTPIVDDPYS
FGQVAAANAISDIYAMGGKPLTVLNIVAFPIKTLDKQVLTDILRGAADKVQEAGATLVGGHSIDDNEPKFGLAVTGLIHP
DRVRTNAGAKPGDKLILTKPIGVGILTTSIKKDQLSEEEVLRVTSVMATLNKTAAEVMDGFEVHACTDVTGFGLLGHASE
VAKGSGVGVRIGAADVPFLPRVRELAEAGFVPGGTKNNFAHVQDTVDFDPSIDQIGQWMLCDAVTSGGLLISAAAADADE
LLARLLDAGVEAAMIGETTADHPGRIAVAKTIR
>lcl|NZ_CP026520.1_prot_WP_042230957.1_1862 [gene=mnmH] [locus_tag=PC41400_RS09565] [protein=tRNA 2-selenouridine(34) synthase MnmH] [protein_id=WP_042230957.1] [location=2101936..2103006] [gbkey=CDS]
MFRDITVEQLLELQEQRGIQLIDVRSEGEFEEFTIPGSRNIPVFDNGERKEVGTIYKQVSVQAAKEKGLEIFSAKLPSFI
KQFEEIPGQKAVFCWRGGMRSKTAATVTSLMGIQMYRLSGGIRSYRKWTHEKLETFEFKPVCIVIGGPTGSGKTELLQKL
KEAGYPVLDLETMAQHRGSIFGQIGLRPSNQKTFESRLVHELMEVNEQPFVLVEAESKRVGKVVMPDFLFNAKETGIPLF
LDIPMERRVANILADYQPSLHKEECIDAYQLIKKRIHSPIAAAIEQYLKEDRFGEAVALMLEHYYDPRYAHAAQNYEQET
VTLQAADTDEAFRLVTAQLKERFPGYRPKIQSSNAV
>lcl|NZ_CP026520.1_prot_WP_042230956.1_1863 [locus_tag=PC41400_RS09570] [protein=DUF948 domain-containing protein] [protein_id=WP_042230956.1] [location=2103197..2103718] [gbkey=CDS]
MTWIELSAVCAAIAFAILVWQAVLALQAFRQSLIRLEEALGKSRQRLEETADRTAELLEETRELAAETRSHLRTARSVLD
GAAKFGTALHEGAEAVGGAASALAESVLKVQQAVHTRQNRIVEAADWTAAGIELWRRWRKSGSEAAAGSSGPSPDSGAPP
TARRENGVSPGGL
>lcl|NZ_CP026520.1_prot_WP_042230955.1_1864 [locus_tag=PC41400_RS09575] [protein=YtxH domain-containing protein] [protein_id=WP_042230955.1] [location=2104073..2104414] [gbkey=CDS]
MANSTRTKDLVFGALFGSVVGAVTALLFAPKSGKELRSDIAAQAANVSEKTQEIARTVGEKTSELVGKAKEVTLNAAGGI
RSIKSGESQIPGKLAEEEIAASEAAQDLVILGK
>lcl|NZ_CP026520.1_prot_WP_042230954.1_1865 [locus_tag=PC41400_RS09580] [protein=response regulator] [protein_id=WP_042230954.1] [location=complement(2104546..2106171)] [gbkey=CDS]
MKALIIDDEAHVRETIRLLVDWGQYDITEVLEASSGEEGIRLITAEKPQLVFTDIMMPGLTGLDVLSRIAESGIPTKAIV
ISGYDDFTYVRSALQFGSFDYLLKPIDEEELEEAVGRAVAAWRSEENERAQRLQSETKANLYKPMLADKSFVQLLLDPEQ
YPLHAETLEREFGIGRDQKQVLAAVVSIQPLDARLQAKFESAVDLLLFAVTNICSEFMSKSNLGLAFRHWHRHAAVLLFF
DRFHQAGAILEEINGGLYAALRGHFHLGYGTVQPFPSGIQRTFHEAETALQLRNLREQDIFLHPYHVRRQALGTLQAPLA
AREDELTLAVRMNDLAGIRRVLEQWELQIRKLPYIHTRQLEMWWQEFTLLHSRLLLENEVDEDGEEKLSAADDLSFCLPL
DEHGKPDLAIWTKRMESILTGIAQFLLRGDRNSVIRSIEKYVRLHYREQILLDDLSERFHLSPSHISRSFKTEFGENLSD
YVTRLRINKAKLLLADPELKINTVAEAVGYQDEKYFSRAFKKSEGISPNLFRKRNTCPTAD
>lcl|NZ_CP026520.1_prot_WP_042230953.1_1866 [locus_tag=PC41400_RS09585] [protein=histidine kinase] [protein_id=WP_042230953.1] [location=complement(2106168..2107985)] [gbkey=CDS]
MRSRLRNKLIAFILAATILPIATSILVSHFYTQSRLEDKAIRDNLNLIHQGKLNLDTYLQSVEQNMKLLYHNTVMMEILE
NRTDNFFSESRLFLTEQEVLRSMQTLKYASQDVRQVYLYVEQARKSYLLSENRLVRTDQGAAPFKPDIRREESRITPPHP
MSAYGKQFFIYSSGDVFTYQTALTNIVDNIPLGSLSLDFSPDTIKAICDELYTKGSEELYIIGPSGEIMYQAHPDPQGGK
KPVPWLQELNQSAARAERGYFRSEDASYNGIHIYETIHAPGGSWLLVKGVPDSVLYKSQRELTGIWTLVGVIFLGLATVV
TIYVSLRMTAPISALIGYMNKIESGNMNVDIRVKADDELGILARRFRSMMHTINELILREYKLEVANKTNELKALQAQIN
PHFINNALQSIAARSLQKGDREVYSLISSLGKMMRYSMKTEETIVPLKSEWDYAKSYIKLQKQRFRDQFEAEMELEPGLE
SILVPKMVIQPLVENYFKHGFDGSRSDNIVRIVCRSTGDRSLLHIEVSDNGFGIGEDKLTALRADLEACMAEMGEPGNHI
GLANVLSRLKLYYREQTTMEIGRGEDGGFRVLLTIPIMQEEGEPE
>lcl|NZ_CP026520.1_prot_WP_042230952.1_1867 [locus_tag=PC41400_RS09590] [protein=sugar ABC transporter permease] [protein_id=WP_042230952.1] [location=2108224..2109108] [gbkey=CDS]
MTTRSKTNDLLQQWVFVGPALLFFTLIVVIPFLMSLYYAFTEWNGVSEGVKFIGLDNFKRIIFEDTDFHNAFWFTIRFSV
AMIILTNLVGFVLALLLTKAFKTRNVLRTVFFMPNVIGGLLLGFIWQFIFVKGFASIGEMTGISFFNLPWLGDAPTAFWG
IVIVSVWQGAGYLMVIYIAALANVPKDLMEAATIDGAGRWQLLRSVTMPLIMPAFTVCLFLTISWAFKMFDLNLSLTKGG
PFNSTESVALNIYQEAFRNNRLGLGTAKALIFFFVVALISLIQVSYTKRKEVES
>lcl|NZ_CP026520.1_prot_WP_042230951.1_1868 [locus_tag=PC41400_RS09595] [protein=carbohydrate ABC transporter permease] [protein_id=WP_042230951.1] [location=2109105..2109941] [gbkey=CDS]
MSSGSKYTSGTFILEVLTALLGILFLVPFYFVLVNSFKPFAELLQNTASLPKSFSLDNYSRVVDIIHFPQVMGNSLMITV
LSNIGLVIFASMSAYRFVRKPSKLNNVLFLLFVAAMVIPFQSIMIPLVKVANNLGLMDSRLGLVVCYFGFGVSLNVFLFH
GFIKSIPKEIEESATVDGCGWFSIFWRIVFPLLKPIAVTVILLNSLWIWNDYLLPSLVLTNTKLHTIPLATYAFFGQYTK
QWDLAMAALTLGVLPIIIFFLLLQKHIIEGITAGSVKG
>lcl|NZ_CP026520.1_prot_WP_042230950.1_1869 [locus_tag=PC41400_RS09600] [protein=ABC transporter substrate-binding protein] [protein_id=WP_042230950.1] [location=2110119..2111432] [gbkey=CDS]
MTTNKKWIAVGLTTALVGGLLAGCGSKDQGQGGVSPSGSPAAGGEKKNVTLKMFQFKVEIAEALSKLIAEYEKETGVKIE
VQTVGGGADYGAALKAKFNSGDKPDIFNNGGNSDLDVWQEHLEDLSDQPWVKDLAQGAGDPISKDGKIYGMPVGLEGYGY
IYNKDLFQKAGITELPKTLTQLEDAAKKLQAAGITPFVNGYGEWWVLGNHFVNVPFANQPDPDKFIKGLNDGSQKIPGNA
VFDNWVKLFDLTLKYGNKNPLQTDYNTQVTEFATGKAAMTQQGNWTQVQISKTNPNINIGFLPMPISDDAAANDKLLVGV
PNNWVINKNSPNKDEAKKFLNWLVTSDMGKKYITNEFKFIPAFKTISVDEKVVGPLAADIMKYAKDGKTLSWNWFKFPGG
EATSKKFGDTMQGYVGKQLNKDQMLQDFQKAWDSLKK
>lcl|NZ_CP026520.1_prot_WP_042230948.1_1870 [locus_tag=PC41400_RS09605] [protein=GNAT family protein] [protein_id=WP_042230948.1] [location=complement(2111679..2112254)] [gbkey=CDS]
MNRTPPLSPGGRFETGRLYLRLLAPEDALALLDYRQRNRAEHAPWEPVRGEAFFTAEAQQAAIRGFEDESATDRAYSFGV
FHQPDGLLIGTVRLSHIGRGPFQNAYVGYSIDAAYGGRGLMTEALRECLRIAFGPLGLHRVQAAIIPRNVRSLRVAEKCG
FVRIGLSPRYLNIAGVWEDHILFALTSEDFG
>lcl|NZ_CP026520.1_prot_WP_042230946.1_1871 [locus_tag=PC41400_RS09610] [protein=AraC family transcriptional regulator] [protein_id=WP_042230946.1] [location=complement(2112364..2113224)] [gbkey=CDS]
MLKPFHSVSVNPHNGKGELFVLFAGYGQTAPCHEVGPQMLDYYLIHTVESGKGRFSCLGKEYELGPGDTFVIFPGEVVTY
ASDKRDPWRYRWIGFKGTSADDLLIRLGITAQRPVVHAEGLPRLPVFLRRAERALREAGPACDLQAEGELRLLLAKLASH
ADVPSRRGEETDAGRQQIDRAVRWLQLQYAQPVTIEQLAQHMGYHRTYLSKLFRQHTGLTPTQFLLKIRMERAKLLMTEG
STIEQIAYSVGFSDPLYFSKQFKKWFGCTPSEYRGKHGRLAYDCSR
>lcl|NZ_CP026520.1_prot_WP_042230941.1_1872 [locus_tag=PC41400_RS09615] [protein=disulfide oxidoreductase] [protein_id=WP_042230941.1] [location=2113329..2113763] [gbkey=CDS]
MKSNSFLKNNALHMAWAISVVATLGSLYFSEIMDFTPCKLCWYQRILMYPLVLILGIAATRKDNKQVIYALPFSVWGMGI
SLYHYLMQKTSLFKEAASKCGIVACDQDYIDWLGFITIPLLAFTAFTLITVILGLLWVQMRKER
>lcl|NZ_CP026520.1_prot_WP_042230939.1_1873 [locus_tag=PC41400_RS09620] [protein=alpha-glucosidase/alpha-galactosidase] [protein_id=WP_042230939.1] [location=2113911..2115212] [gbkey=CDS]
MSKITFMGAGSTVFAKNILGDCMVTEALQGYELALFDIDGERLRDSETMLNHIKNTVGSTCRIKAYSDRKEALRGAKVVI
NAIQVGGYDPCTITDFEIPKKYGLRQTIADTVGIGGIFRNLRTIPVVLDFARDIREVCPDAWFFNYTNPMAVLTNVLNTH
GGVKTVGLCHSVQACVPSMFRQLGIDPEGVQSKIAGINHMAWLLEVTKDGKDLYPEIKRLAKEKQKEKHHDMVRFELMFQ
FGYYVTESSEHNAEYHPYFIKRNYSELIEKFNIPLDEYPRRCVNQIERWKNLRAELVNNPQLEHKRSGEYASYMIEAMET
NVPYKIGGNVMNTGLITNLPKEACVEVPCLVDAGGITPTYIGDLPPQLAALNRTNINTQLLTIEAAVTRKKEHIYHAALL
DPHTSAELSIDDIVALCDELIEAHGDWLPAYTS
>lcl|NZ_CP026520.1_prot_WP_042230937.1_1874 [locus_tag=PC41400_RS09625] [protein=ABC transporter substrate-binding protein] [protein_id=WP_042230937.1] [location=2115581..2116843] [gbkey=CDS]
MRKFTAGMLAAVLASAAALSGCSAVGGGKTELNFWYGWTGPEAEALEKLIKEFNDTHDGIKVKGLSQSDYQKQLTAITGG
NPPDLASQFGQNVASWGVKGAMIPLDAYIKKDQVDLADFIPSALSTSQYEGKTYALPIAMHFSMLYYNKDLLKEAGFDGP
PKTIAELKTYAQKLSVVQEGGRLDRLGLWPGADAYTFTQVFGGKFWDPDKKEVTPDDPGFKEAILLSKELWDKYGTENLD
RFSASLGKYQSSQNPFFSGKYAMALDGEWLPTFIKQYAPGLNYGIAPIPYNEDKPELKNAGSIATSVFYIPKGAKHPDQS
WTFLKWLTEKQQMVKFTAALGNLPTRLSSLDDKTYDSVPGFKEYTEYAKSPNLKSFPTLPFMNEYMDEITKKYNDILRGK
VGVDEGLRQIKDKIQPLVGK
>lcl|NZ_CP026520.1_prot_WP_042230935.1_1875 [locus_tag=PC41400_RS09630] [protein=sugar ABC transporter permease] [protein_id=WP_042230935.1] [location=2117023..2117967] [gbkey=CDS]
MSTDTLNAGTGRPTRSRKARRDFWVGTAFALPWIVGFLAFTVYPFFGSLYFSFTSYDLFNSPQWVGLQNYTEILKDETFY
KSLRNTFFMAFISMPIGLLSSLLIALLLNVKVKGIAFYRTVYYLPAVIPVVASAILWTWLLNPDYGLINMLLRSLHLPDP
AWLLDPRYTKPSLILMGLWGSGAGSLIFLAALQGIPQQFYEAASIDGASVWRRFWHITLPGLSPVLLFQLIMGLIGSFQI
FTESFILTGGNQGGALGGPDHSLMFYAVYLYQTAFMYLKMGYASALAWILFVIVVLMTVLVLKTSARWVYYGGD
>lcl|NZ_CP026520.1_prot_WP_042230934.1_1876 [locus_tag=PC41400_RS09635] [protein=carbohydrate ABC transporter permease] [protein_id=WP_042230934.1] [location=2117968..2118882] [gbkey=CDS]
METSTNLQGSGLNVLPDSPAGMPKNRRRRPLLVLRRTILPHAALSLLGLIFLFPFIWLLLTSLKNPSEIFQVPPAFFPEK
WQWSNYGNAIGSIPFFLYMGNTLLLCAVNIAGQLFSAPLVAYSITKIPWRGRGIIFSLVVATMILPAQVQMIPVYIIFAK
LGWINTFLPLTIGSFFGAPFYIFLLRQFMMGIPGELCEAAKMDGASELRTYGQIIVPLLKPPLMTIALFTFVGTYTDFMG
PLIYLNDNTKWTITVGLQGFLQDHGAQWELLMAASAIFTIPMVVLYFFGQKYFMKAGSTLSGFK
>lcl|NZ_CP026520.1_prot_WP_241688833.1_1877 [locus_tag=PC41400_RS09640] [protein=sugar phosphate isomerase/epimerase family protein] [protein_id=WP_241688833.1] [location=2118970..2119824] [gbkey=CDS]
MAMKISLSMWSVHTYWYGGKWDVTDFLRFAAGTKAEGVELLSVFWKDRERELPEIDRVLDELGLEVACFCACNNFVSADA
AERAAQLREVKEAVDMAVHFGAPVVRVFSGDRPDETMPVDEGMGYVLEGLKAAAAYAEEKGKVLCLENHGLFAGRSDQVL
AIIREVGSPALLGTFDTGNFLLVGEEPSAAVQELRGRIGHVHVKDFLRTEPGAEGAMNALDGTPYVGRIAGEGDVDLPYI
FRELAASGYDGWLTVEFEGLEEPAEGSVKALDYVHEQVELLKRE
>lcl|NZ_CP026520.1_prot_WP_042230933.1_1878 [locus_tag=PC41400_RS09645] [protein=Gfo/Idh/MocA family oxidoreductase] [protein_id=WP_042230933.1] [location=2120266..2121312] [gbkey=CDS]
MSIRVGVIGTGSIAHMFHLKHYKDHPDVELAAVADIDKDRADKTAAQYGTASYASAEEMLNAERLDAVSVCTFNDSHVAL
SLLALERGLDVLVEKPMAMNRDEAARLRDKAAGSDRLVMVGMSHRYRADVQALQGMVASGGLGSIYYAKTKILRQRGVPE
GWFTDSRYSGGGPLMDIGVHVLDLAWWMMGCPQPESVTGKLFRKIAPYTTKGIAAYTAHSDINKSEGVYDVEDLGTAFIT
FTNGAVLTVEASWAVNGSQDDAVKVELYGTGGGGSLEPLSIFTEVNGFPVEHKLSVPSVDYYKAEIAHFIDCVKTRRQPL
SDAAQGCEVVGMLEAIRLSSESGELIRL
>lcl|NZ_CP026520.1_prot_WP_042230931.1_1879 [locus_tag=PC41400_RS09650] [protein=acyltransferase] [protein_id=WP_042230931.1] [location=2121330..2122523] [gbkey=CDS]
MNEPKITYLDGLRGAAALVVVFAHFLQVFLPSMFEARPQVSHFAFDQWIPRTPLNLLFNGNFAVCLFFVLSGFVLSRRFF
RYRNRSIVTAALLKRYFRLAVPAFCSLLLAFAVMKLDWNFYEEARGYTRATMPDPYTTVPSFGTLLEQGFVHTFFGYGMA
YNPVLWTMTYELFGSFLVYGLLLAFGRSRFRFILYAAGGWLFRDSYYLGFVLGMLLSDLSAGSGRNLIVLRPGAVPVVLA
LLGLFLGSVPYVGWQGTFYAWLPLGPGGFPGVVTYRTAGAALVLLALLNSPSGQKLFGGRSMAYLGKLSFSLYLVHFAVL
CSFSCLLFLRLDGWLPYGLNILVTTLVSLPLLFALAHLMYRLIDEPVVRLLSRVRFAARGRRNTGAAVRHAGTEAEL
>lcl|NZ_CP026520.1_prot_WP_042230929.1_1880 [locus_tag=PC41400_RS09655] [protein=DUF4901 domain-containing protein] [protein_id=WP_042230929.1] [location=complement(2122791..2124248)] [gbkey=CDS]
MLIDELKQKAAVTGSVPAHFDLIIEDLQEQEDGETQALFVWQEPSGESGISVTLDAAGRLLSYSVDEELCKEAAPIGEEA
LRERARSFFLSQHPEDADRFLLTESRFEERSAFFEWAQTALDLPLPQTGAFIRLTLSGEVTSFTYKGSAAAPPLPVSLVQ
PEQLRQQLAETVRMSLAVVKLIPALYAGGDDRLRLVYQPEPGYRRFDAVTGLAQGTDAEAEEEHEAYKAYELSPEETGDP
LSSLEELIGISTYPLVKDRERDMDENLRGIVWHAADREENPADLSLDGYFRRRMEDTVKATADKDTGRLRSFIWMVRRTG
KLALSYEDCLERAEAFLRKAMHDWLQVPGLAFYFQEKSAEDAGSESEGQLRYFFTWQLRYEDVPLYLQFVTLCVNASTGY
VDRFSGPDWNPEEIRNLAPAAKLNADEAHRLYVEAVDFKLEWDRDYDAETPEGAYNLLYTPILKGSGRNRIAFIDAASGE
FVTQA
>lcl|NZ_CP026520.1_prot_WP_042230927.1_1881 [locus_tag=PC41400_RS09660] [protein=polysaccharide deacetylase family protein] [protein_id=WP_042230927.1] [location=2124408..2125451] [gbkey=CDS]
MKKTAAASAVLLLSGILAAGCTGNSNVSNSGSPVQSPAATAGASTSPAAAPSPAPEKTPAGSAKPSASPSASASPGISPA
PSSAVKASPSAAPSAAPSAKPSASPGAAKPSAQPKGESSASRQALSWYYMKKPKGQVPDFPKETKSFTAKEKAVWVGQGK
NVYLTIDNGGPMGDTAKLLKTLKENNVKANFFISGNNVKAHPDFIKQLAADGHLVANHTMTHKDMNTLSDEQVRKEITDF
EKLYKSITGQDVVKYFRFPYGKYSTHLLDLTSDMGYTSVFWSTAMRDWEPRANGAEDPYNDIMNNLHPGNIILMHQGSKE
NMEALDRILKDVKKAGYSFGLVSEIKR
>lcl|NZ_CP026520.1_prot_WP_042230925.1_1882 [gene=racE] [locus_tag=PC41400_RS09665] [protein=glutamate racemase] [protein_id=WP_042230925.1] [location=2125664..2126473] [gbkey=CDS]
MQQAIAILDSGVGGLTVVKEVMRQLPQEKVIYFGDTARTPYGPRPADEVLTFTNQLVDYLIQFNPKMIVIACNTATAVAL
EQIRDRLDIPVLGVISPGARAAIKTTRSGIVGVIGTEGTIRSEAYEQALKTISPTIQVHSLACPALVPLVERGLYFSEEA
VDTVRDSLQPLLDKPMDCLILGCTHYPFLTEAIARVVDPEVTLISSADETAREISTILYHRGMLATTKDIPVHQFFCSGD
AEMFHLITGSWLREQLLGTPVVWQVPTFV
>lcl|NZ_CP026520.1_prot_WP_042230923.1_1883 [locus_tag=PC41400_RS09670] [protein=M14 family metallopeptidase] [protein_id=WP_042230923.1] [location=2126683..2127885] [gbkey=CDS]
MPFRHVVQQGETLQRIARLYAVNSVSLLAANPQLSLHEYVIPGQMILIPAKLDHQYTVQPGDAWESVSLRFGLSREELWK
ANTFLDPATLRPGVTIDVPAVRDRSVVRTAEEYGYPEMLRDLTLLKERYPFLEETCIGYSVLDRKIPAVRIGTGSREIQV
NASFHANEWITSLAAVKFIEDYAEAVRQGGDIHGSDAAALFRETSLWIVPMVNPDGVELVLGGVYPAHPLFASLHEWNGG
SFDFRRWKANINGVDLNDQFPAFWEEERARRAVPGPGPRDYTGEAPLTEPEAIAVADFTRSRSFRHVVALHTQGREIYWN
YRDYEPDDAEFLALKLQQASGFKSVKLRGSDAGFKDWFIYEFRRPGFTVELGSGINPLPLEQFPALYEETARLLVRALAL
>lcl|NZ_CP026520.1_prot_WP_042230922.1_1884 [locus_tag=PC41400_RS09675] [protein=hypothetical protein] [protein_id=WP_042230922.1] [location=2127986..2128195] [gbkey=CDS]
MAGKSGKGKGGKKIESSGRVQYTMRIVESSTCAVCKQQCARGLRYLERMSQPGALGKGVPCILTRRKLT
>lcl|NZ_CP026520.1_prot_WP_042230920.1_1885 [locus_tag=PC41400_RS09680] [protein=hypothetical protein] [protein_id=WP_042230920.1] [location=2128452..2128967] [gbkey=CDS]
MSGILPLDQFKELEREEQIRLINQWKEDYSVGEITSAWKLNNAATYYTILKKLNIYNDTVRTSNKFMGYTNYRSQSRSLE
RPSEQSDFRRLPESVQAENRPAVSAPALSISDNFRYNVNGTYPAEEVAQILERLSLFLKIRSHVPVNVELNIRFGDSDAV
TSELQSGEPHS
>lcl|NZ_CP026520.1_prot_WP_042230919.1_1886 [locus_tag=PC41400_RS09685] [protein=hypothetical protein] [protein_id=WP_042230919.1] [location=2129205..2129438] [gbkey=CDS]
MKKLLSLRHWSHVFKRAGKLLMSPQVPIGSKLLFVVPALLYWVLPDVLPFMPIDDIAVTMMLASWFTGWMERKYPQA
>lcl|NZ_CP026520.1_prot_WP_042230916.1_1887 [locus_tag=PC41400_RS09690] [protein=hypothetical protein] [protein_id=WP_042230916.1] [location=complement(2129688..2129891)] [gbkey=CDS]
MIENLESNYDCAHAGLDLHQLKEELNDLRISMIPGSLVHNEKVNRLENQIRFIENKCSIHHGDSGSL
>lcl|NZ_CP026520.1_prot_WP_042230914.1_1888 [locus_tag=PC41400_RS09695] [protein=hypothetical protein] [protein_id=WP_042230914.1] [location=complement(2130209..2130388)] [gbkey=CDS]
MDANLHIPQGDDTILVEMTVKEAMALSGQKFHSDHKVETEAIKKVKKSLENRLISGVSQ
>lcl|NZ_CP026520.1_prot_WP_042230912.1_1889 [gene=lepB] [locus_tag=PC41400_RS09700] [protein=signal peptidase I] [protein_id=WP_042230912.1] [location=complement(2130601..2131155)] [gbkey=CDS]
MNKVAKEIGGWALSLGIAFVIALVIGIFGIQPTKVLGHSMDPTLSDQERIYVSKLSHTLGYEPDYGDIVIIDSRVNRERT
LKDDIMEHPLFSLLSGKVDDNLYVKRIIGKAGDKLQFKDNKVYRNGQPLDEPYLNETMLYQSDEEITVPAGHVFVMGDNR
NNSKDSRDIGPVPLDHVLGIKING
>lcl|NZ_CP026520.1_prot_WP_042230909.1_1890 [gene=ytvI] [locus_tag=PC41400_RS09705] [protein=sporulation integral membrane protein YtvI] [protein_id=WP_042230909.1] [location=complement(2131292..2132398)] [gbkey=CDS]
MISFYKRYYKTILDIGLIVLTVFLFMKLFSFIYAIATPIFWAFIIFAMIEPLARFLHRRGLKKMLATTISTLLFVLIILA
VFTGAGFLFTSQIDNLVTVLPKYTQMLQEQITKRTIDLQNSIEALPPDVIEKSREIANQMATKATSFATNFLKSLLNFLT
SGVMGLVTSFSTFMLNFVAGVILAFFLSLEIDWWKKNAREKTPKTFRKAFDFLRENVIKGIVSYLKSQLILISITFVMVL
VSLMILGVNNAFTISLLAAFFDVLPLLGVSTLFVPWIVYHFIAGSTMMGIWLTVVLAVVVLVRQVLEPKITGESLGVSAF
TMLSFMIISLSLFGVAGVILSPILIILIKALYDQGYLRKWIRSPEDEY
>lcl|NZ_CP026520.1_prot_WP_042230907.1_1891 [gene=map] [locus_tag=PC41400_RS09710] [protein=type I methionyl aminopeptidase] [protein_id=WP_042230907.1] [location=complement(2132575..2133321)] [gbkey=CDS]
MVTIKTKDEIEKMHRAGKLLAACHREIAKMIRPGVTTWEIDQFTEQFLRENGATPEQKGYKGYQYATCASLNDVICHGFP
KKEPLNNGDIVTIDMVVNLDGWLADSAWSYAVGTVSEEAGNLLETTKESLYKGIEQAVIGNRIGDIAHAIQVFAEERGYS
VVRDFTGHGIGQKMHEEPQVPHYGPAGRGTRLKDGMVLTIEPMLNTGMYFATVDLDGWTARTVDGSLSAQYEHTIAITQD
GPLILTAQ
>lcl|NZ_CP026520.1_prot_WP_042230905.1_1892 [gene=fosB] [locus_tag=PC41400_RS09715] [protein=metallothiol transferase FosB] [protein_id=WP_042230905.1] [location=complement(2133641..2134075)] [gbkey=CDS]
MTIRGINHLLVSVSDLERSIRFYREVLGARLLVKGRSLAYFDLGGLWLALNTEREIPRNEIRQSYTHLAFTVDEKDLPAL
EAKLAACGALLPGRERDSRDKSSLYFEDPDGHKFEFHTGTLQDRLDYYREDKGHMRFFDDPMVE
>lcl|NZ_CP026520.1_prot_WP_042230903.1_1893 [locus_tag=PC41400_RS09720] [protein=heme biosynthesis protein HemY] [protein_id=WP_042230903.1] [location=2134184..2134486] [gbkey=CDS]
MNVKITRNAAKVLKLELEKEENKGLLVRVQVSHNHGDHAHYALGLDTQTDNDVVVSTDKDIDVLLEKDEPLLDGIRIDYF
YIPQEGFMITNPSKGNHGDH
>lcl|NZ_CP026520.1_prot_WP_042230901.1_1894 [locus_tag=PC41400_RS09725] [protein=DUF1450 domain-containing protein] [protein_id=WP_042230901.1] [location=2134486..2134707] [gbkey=CDS]
MANDIRICDKCKHMRVKTALPKLQKLASDAEVKVGCKSYCGPCARYAFVFINGRYIKAPTEDEAIEHAKKFLK
>lcl|NZ_CP026520.1_prot_WP_042230899.1_1895 [locus_tag=PC41400_RS09730] [protein=TIGR01777 family oxidoreductase] [protein_id=WP_042230899.1] [location=complement(2134771..2135676)] [gbkey=CDS]
MRIAISGGTGFIGGHLIAHFKKRGDELILISRSASGHAYAGVQTVTWSEIEHSPRYLEGIDAWINLAGSTINQRWSETAK
RQILSSRLKTTAAVASALKALQNKPPVVVNASAMAIYGTSETHTYDEFSPARADDFLAEVVKEWEEAADRIPAERIVKIR
TGLVLGTDGGAFPKMALPFKFGAGGRIGSGSQWMSWIHIGDMVALIDACIRNEDISGPVNATAPHPVRNREFAQTLGSVL
RRPHALPVPAFALKLVLGELSMLLLEGQRVLPRKLLEHEFQFRYPTLEEALLDLTGAERPA
>lcl|NZ_CP026520.1_prot_WP_042230897.1_1896 [locus_tag=PC41400_RS09735] [protein=GNAT family N-acetyltransferase] [protein_id=WP_042230897.1] [location=2135912..2137024] [gbkey=CDS]
MIVIRKLSECTLDQAVQVWNAGFSGYLVDMTTNPDAFMTRQAYEQLSPQHSIVAFDGEQPVGFVLNGIRMAGARKISWNG
GTGVIPAHRGRGVGRKLIEASLDIYRKEGVDTATIEAFASNVPAIALYESVGYVTRDTLLVLRKAGGAAGGGPAEGTGES
RGALGDALLPAAGETGGLPDAAAVRAGGLANASGGLAATSRQADASQEGRGGHAALSAAEAGHADGSQSAAGRYTLRHGT
ARDAGLLPFYPALGAWQVHWASLTAGESLIMLQDGKPAGYALYKRVYEASGKLAGVTLYQCEASPDNREPETVLRTLLEA
VLDGYGEVSFSASHIPSSRTDLLRLLEEKGFQTVHELVWMTRDMTEEGRA
>lcl|NZ_CP026520.1_prot_WP_042231183.1_1897 [locus_tag=PC41400_RS09740] [protein=aminopeptidase] [protein_id=WP_042231183.1] [location=2137152..2138099] [gbkey=CDS]
MDRSLIEISKSLLQSSLGVKPGELLVVVADDYKKELAEHVHQAGLELGADASLVVMKHREKSGQEPPAPVAAAMREADVA
VCLTQHSLTHTGARKEAAASGTRVATMPTITEDMFREGAISADYAKVRELTERVTARLTAGSRVKIEKEHTALTFSIAGR
AGVPSTGVYLNPGESGNLPSGEGYIAPVEGSASGTLLVDGSIAGLGKVTSPLTLTVENGRLVGADGEQGEALLAVLGDGD
GRLLGEFGIGTNDKARITGVVLEDEKVYGTIHVAFGSNNTFGGTVAAGVHIDLMVLSPRVYIDDELILENGRLLV
>lcl|NZ_CP026520.1_prot_WP_042230896.1_1898 [locus_tag=PC41400_RS09745] [protein=MFS transporter] [protein_id=WP_042230896.1] [location=2138417..2139643] [gbkey=CDS]
MNRIRHFLSSYHPLVHTLLFGTVMARAASSMSMPFLALYLVATTDMSKGMIGLVIGASSLAGTVGGFIGGTLSDRIGRKA
VMQGALFTWAAVFVGFSLVTHPLLFLLLNLISGLCRSFYEPVSQALMADLTEKEKRLQVYSLRYLAINVGVAVGPLLGAA
FGMVKGSLPFLLTGIVYLFYGIFLVVMMRKFKVQTVSKEVKERVTFAAAWRVVRKDMILRYQLLGGMIGAIGYSQMAITL
SQYVNENFAQGVQLFAAMMSVNAVTVIVLQIPLSKLADRKSPLTAIVVGNLLFALGDVGFALSGHWLLLIASMVVFTIGE
TLNFPASTMLMDKIAPDHMRGTYFGAQTLTSLGHFIGPWVGGLLLASIGGNMFLVMAVVTVTSSFLYWRGSQLLTKRIEG
EYAATGSA
>lcl|NZ_CP026520.1_prot_WP_042230894.1_1899 [locus_tag=PC41400_RS09750] [protein=hypothetical protein] [protein_id=WP_042230894.1] [location=complement(2139820..2140062)] [gbkey=CDS]
MQGILALVLFFISMLMFAALVHSKSKGQNQRFRFLHHPLFRKTAVVVSVLAFLLAVVVSPKADEGPHTEAKLTTAAVTVR
>lcl|NZ_CP026520.1_prot_WP_042230893.1_1900 [locus_tag=PC41400_RS09755] [protein=class I SAM-dependent RNA methyltransferase] [protein_id=WP_042230893.1] [location=complement(2140458..2141615)] [gbkey=CDS]
MAKLDLIATCPMGLEAIVAREVKELGYEDVTVENGRVTFTGDELAICRTNLWLRTADRILVKMGQFEARTFDDLFEGTKA
LDWPAWIPEYGEFPVEGRSHKSQLSSVPACQGIVKKAVVEKMKERYRTEWFEEDGPRYVIEVALLNDIATLTIDTTGPSL
HKRGYRKLVTEAPLKETMAAALVNLSRWKPDRPLYDPFCGSGTILIEAAMLGWNIAPGLRRSFPSEYWDIIGQSRWEQAR
EEAYDSLNDDIPLEIAGSDIDPKAIEVAEAAAKAAGLSKELKFYVEPVAKAKPRSDFGCLITNPPYGERLGEQKEVERVL
RQLGQLTGSLHTWSSFVLSPSKQLEHYFPRPADKRRKLFNGRIECQLYQYFGPLPPRRRNTEPVE
>lcl|NZ_CP026520.1_prot_WP_042231182.1_1901 [locus_tag=PC41400_RS09760] [protein=O-methyltransferase] [protein_id=WP_042231182.1] [location=2141899..2142564] [gbkey=CDS]
MTDQTNGGSEKTAPGPEEYLEALYAEDEALGLVKEAIRANNMPDISVAPGYGRLLTMLVGLTGAKRVLEIGALGGYSAIC
LARGLGDGGQVLSLEVNAAFAETARQNVTKAGFGDKIGYRVADAVESLAALKEEGAKFDFFFIDADKVNYPVYLEHAIEL
ANPGAIIIGDNILMRGRTIDAAVTKKSVEAMREFNRRFASDPRLESTMLPAYDGLAVARVK
>lcl|NZ_CP026520.1_prot_WP_042230892.1_1902 [locus_tag=PC41400_RS09765] [protein=MFS transporter] [protein_id=WP_042230892.1] [location=complement(2142747..2143958)] [gbkey=CDS]
MINWKVNLAVLWVGQFFVMAGMTMIMPFLPLYLKEMGLQDAQEIATWTGIIFAGNFVTSFLFQPFWGGMADRYGRKVMLL
RSGFGMAAVMTTMGFAHAPWVLLLLRLVNGTISGYAPASVALMSTNTPREHIGFAMGTLQSGAVAGSILGPFIGGLLAEL
IGFRYIFYVTGAMLFIASMVTMLLVKEKFNAAEAKKQPKISVIQGYKDLRPIRQLPTLYSITFVIQFSIQCSMPLMALFV
EELHGTGELLAFYAGLVASITGFSNMVAAPLLGKLSDRIGPQRILFVSLIGAAIFFIPQAFVQNVWQLMAARFCLGIFIG
GLLPTVNSLIRTYTPVGMESRSYSFNTSALALGNMLGPVIGGALSGFLNIRELFLFATVMLAINAAWSYRTLLGKHKISD
GAL
>lcl|NZ_CP026520.1_prot_WP_042230891.1_1903 [locus_tag=PC41400_RS09770] [protein=spore coat protein] [protein_id=WP_042230891.1] [location=2144209..2145483] [gbkey=CDS]
MPFGAHETLEVHEVLTEKINMIQHFSFYARQARHPQVKAIAEHHLQAGVQHYNELVAYTHDYSAARGHAPYGQPPFVSVQ
DILYGLRHPAPVSPETDGSFNDEQILSAILCCHKNSAKNQMTAALECADPNVRRMLLTAADAAADHAYEVFLFMNSQGQY
QVPTLHDHTAKTYLHTFQPVRESDFGQITGNYGQGYGYAQEFGYAPDYGQNHLYGYPGAGGSQPHQQSHQPQGYQPQNHQ
PGAYGYNGGYNAQPDNGGYSAYGRTETGYGTHSYGSSAYRGQGQQPDAPGFAQPSSYTGGGASGYGSAYGYESAGQQALY
QTQSPSYVQASNSTYQGERNYTPQTGSGTQPHSGTQQHSSTQTHSGAQQHQSGYAASAAGTRQASHAQTPVVNSQRGGQE
EQTDEPDSGALTSKAGDSSSNPIQ
>lcl|NZ_CP026520.1_prot_WP_042230890.1_1904 [locus_tag=PC41400_RS09775] [protein=fumarylacetoacetate hydrolase family protein] [protein_id=WP_042230890.1] [location=2145639..2146265] [gbkey=CDS]
MTGTVRNIYCVGRNYKLHAAELGNDVPEEPMIFTKPTHALAAMEGDLLELPQGKGEIHYEGELVLHISKPYSPGMTVDEL
VDSFGLGIDFTLRDVQSVLKKKGHPWLAAKGFLNSAALSPLRPFPGAAELEKTEFKTVRNGETVQRGNIGELIFPLQTIV
DYVAANYGLGAGDLIYTGTPAGVGACVDGDVFELLWGEEIFGRCTVKL
>lcl|NZ_CP026520.1_prot_WP_042230889.1_1905 [locus_tag=PC41400_RS09780] [protein=CapA family protein] [protein_id=WP_042230889.1] [location=2146312..2147640] [gbkey=CDS]
MKKQRLTLLIVSAIFSAVLVTGLMLVKIKMDAVSAPLAAASSEQPVAKTGGDNQPASSAVPAPEKTPVPAASPVPGKTEE
PAEAAASEPTGNKETKPADKPANRGETKATEKPKGSQDTAPAVRKKVGDHEAVDLIFAGDALMDWSVKETIKKKGPDYPF
QFVKAEVSKADYAFVNLETSVTDATEKDTNQLYNFKSDPPSLKGLKNAGFDLVSIANNHVLDFGRQGFLDTLNHLKQAGL
PYIGGGRNAAEAYKAKTVEINGARIKFLAFSRFIPSRDWFAGPDTPGIAQAYDKSAVLPVIRQESKDADYVFVYMHWGVE
KNHKPEEWQRTFAKEMIDAGADGIIGSHVHVLQGFEFYKGKPIAYSIGNFLFPDYVKDAKADTGLLSLKLINGKITAEFN
PYYIQKDQIVKRDAAYDKKQLDFLQSISYGVKIRGKAIEPAS
>lcl|NZ_CP026520.1_prot_WP_042230888.1_1906 [locus_tag=PC41400_RS09785] [protein=hypothetical protein] [protein_id=WP_042230888.1] [location=complement(2147722..2148054)] [gbkey=CDS]
METNRNKPNEAINPITDENKHNPYTNQKNNVRKYENEDLESVKEHEYHDAKQNSADRDINEVMNGSMAPNMDEIKQLGKD
MEGMKTNEDLTEEGLLPDPIQNEREEEASR
>lcl|NZ_CP026520.1_prot_WP_042230887.1_1907 [locus_tag=PC41400_RS09790] [protein=DUF92 domain-containing protein] [protein_id=WP_042230887.1] [location=2148217..2149128] [gbkey=CDS]
MSGMDVWIGLLGSVLVAGAAYSKRSLTVSGAAAAVVLGTLMYSLGSAAWFGTLIAFFVASTLLSKYKHKRKAAIESGYAK
SGNRDAGQVAANGGLGLLLCIGSVIAPHPAWWAAFVGVMAAVTADTWATEIGGLSRTAPRSILTGRKVAPGTSGGVTALG
LLATCAGGLFIGAAAWLLARLGSGAVTGAADAAAAVAAASGRQAAAVLTAGAQHPLETLLPLLAAGLAGGLAGSLSDSLI
GAKWQAMYRCAVCGRDIERGSHCGSPAEPLRGFAWMTNDAVNVIGSLFGGAAALAAALLAGWL
>lcl|NZ_CP026520.1_prot_WP_042230886.1_1908 [locus_tag=PC41400_RS09795] [protein=lipopolysaccharide assembly protein LapA domain-containing protein] [protein_id=WP_042230886.1] [location=complement(2149282..2149671)] [gbkey=CDS]
MKSNRIQWLLIGALIFALITTVFAVVNMDQVRVNFVFMETNAPLILVIVISTVLGGLIVGLFGILRQVQMQKEIKELRKK
LSGSAVPADTEVLPGTAAAEAASASYEERLPGEEASGKKEFGKNNSGAL
>lcl|NZ_CP026520.1_prot_WP_174446824.1_1909 [gene=pepF] [locus_tag=PC41400_RS09800] [protein=oligoendopeptidase F] [protein_id=WP_174446824.1] [location=complement(2149751..2151541)] [gbkey=CDS]
MNQIPVRKDVPAEHRWKLEDLFANQEAWNKEYNETKEALGKIGQFHGKLSDPAAVKQCFELEDDISLHTERLYVYSNMKH
HEDMTDPTYQALSDKAQKLSVEVSQALSFISPEIVSLSEEQLKALVENKDLAPYKTTLEEMIRQKPHVLSKEEEALLAMA
GNMSGAPSNIFGMINNADMKFPKVKNENGEEAELTHGSYIQFLESRDRQVREQAFKAVYETYAKQKNTIAATLSSNINKN
IFYSQARKYPSVLEMSLFGDNIKPEVYTNLIDTIHESLPLLQRYMKLRKKLLKVDELHMYDLFVPLVDDLKMKITYDEAK
ETIKDSLAPLGKPYVETLEKGYNDGWIDIYENQNKRSGAYSWGAYGTHPYVLLNHKENLNSMFTLAHEMGHAIHSYLSDE
NQPYRDAQYTIFLAEVASTLNEALLMDYLLKHTDDPKKKLYLLTYYADQFRTTVFRQTMFAEFEKITHEKMEKGESLTAQ
EFSKIYYDLNTLYHGKEMVVDKDIEMEWARIPHFYNSFYVYKYATGFSAATSFSKQILDEGQPAVERYLGFLKSGGSDYS
INILKKAGVDMSTPEPIKQAMSVFASVLDQMEELTK
>lcl|NZ_CP026520.1_prot_WP_174446823.1_1910 [locus_tag=PC41400_RS30075] [protein=hypothetical protein] [protein_id=WP_174446823.1] [location=2151754..2151900] [gbkey=CDS]
MGSKFARDYPAIPNLNRDTTTNEKEQTTRISAKMAIGLRNCSGIRLRE
>lcl|NZ_CP026520.1_prot_WP_042230883.1_1911 [locus_tag=PC41400_RS09810] [protein=TerC family protein] [protein_id=WP_042230883.1] [location=complement(2152127..2152963)] [gbkey=CDS]
MDWFSTASLLTLLNIIIIDLVLAGDNAIVIGMAAKNVPKNLQKRVILWGTAGAVGIRVVATVLVVYLLKIPGLLFVGGLL
LVWIALKLLTDKSDHDNIAAKNTVGAAITTIIAADAAMGIDNVIAIAGAAHGHIGLVIIGLLVSVPIVVWGSTIFIKVVD
RFPIVMYIGSGVLAYTAAKMLTEENFIMHDYFSANPLVKWLFIAVVIVGVLVTGYVVKMSGYSVKINDRGQLTLPKHVGH
EANIAPEDRFTVQSDPSGRLVFVKVGPDKDPKSVKSAG
>lcl|NZ_CP026520.1_prot_WP_042230882.1_1912 [locus_tag=PC41400_RS09815] [protein=hypothetical protein] [protein_id=WP_042230882.1] [location=complement(2153285..2153527)] [gbkey=CDS]
MSHVIKKHVIFKRDKWNMVTVEVNGSRLVAREISDQWGEDTHTFLSRPALMQWARERFPRDLFEGTDEEYDAMMEAFRNV
>lcl|NZ_CP026520.1_prot_WP_042230881.1_1913 [locus_tag=PC41400_RS09820] [protein=class I SAM-dependent methyltransferase] [protein_id=WP_042230881.1] [location=2153794..2154510] [gbkey=CDS]
METKRFYEQTGVAMTCRNFEEYVRMFDLEPEKLRGKRVLDTGAGASAFTAEASRREIEARAADPLYAMGPSEIEGLGLEE
IDRFIAKLTERRHMYNWDFYEGPELLKELRKKSFADFAADYANPASREEKYAAASLPELPHSEDEFDLVLCSHFLFLYGE
QFGPEFHLQAMRELLRVCKPGGEVRIYPLLTFGGELYPQLPELLAELAREGVTAEYRPTRLAFIPGSRDYLTLHKVAD
>lcl|NZ_CP026520.1_prot_WP_009675016.1_1914 [locus_tag=PC41400_RS09825] [protein=cold shock domain-containing protein] [protein_id=WP_009675016.1] [location=2154599..2154796] [gbkey=CDS]
MKGTVKWFNAEKGYGFIQVEGGEDVFVHFSAIQGDGFKTLDEGQEVEFEITDGNRGPQAANVTKL
>lcl|NZ_CP026520.1_prot_WP_042230880.1_1915 [locus_tag=PC41400_RS09830] [protein=TetR/AcrR family transcriptional regulator] [protein_id=WP_042230880.1] [location=complement(2154908..2155537)] [gbkey=CDS]
MSSTSRKPPGRPRSSEQAAPVSDRILQIAMHEFIRSGYDAVSMDDIAKASGVTKASVYYYFPSKAVLLTESTIRLMNVVR
AATLKILQKPLPLRERLLEVTYNHLQATTSFDFEGFMNKIETVLTPEQTENMRKAEHQLLEALADAFAEASDKGEIHTAD
SWLAARSYMALLMIGHSKQPDGTFTYPDKRLAADKIVSMLWNGIGNSPA
>lcl|NZ_CP026520.1_prot_WP_042230878.1_1916 [locus_tag=PC41400_RS09835] [protein=MMPL family transporter] [protein_id=WP_042230878.1] [location=2155797..2158040] [gbkey=CDS]
MKSSSFMEGMLSRVAGPRSKWVTVAVWIVLTVLLSALWPGVNSQENNAADQLPADSLSVQAAALEKEQFPASGGVPALVV
FYRQAGLTEGDLGALQKIAGRLTETPLPEQTSVAPFDKLPLPALKQSVSKDGTTVVLPVSFKEKTETKVLKESAAKLKEI
AAAVTGSDPFAVKNLKADGLIARITGPVGISMDANDLFKGADFSLLMATVILVLVLLILLYRSPVMAIVPLVGVAFAYGV
ISPILGKLAQLGWIVVDAQSISIMTVLLFGAGTDYCLFFATRYRTALLEHENKHLAVKEALLGSGGAIGMSGLTVIVSML
ALLFAKYGANHRFAIPFSLAVFIMAAASVTLVPAILGILGRKSFYPFAPLTPEQRAVKEAKRGKPVRGQRAVGRMSETLG
RMVTEKPKTILVVSLVVLIALAGVSSQIKYTFNLLSSFPETMPSREGFTLLSEHYSPGTLAPAKVMVDTEGKKVDAGARL
KALSFVDSVSEPVKGKTNPDIVAYDVLFKDDPFELGSIHHIPDLYAAAEASLQDAGVTSTGSHVRVSGQTAEQYDTENVT
GKDTKVVIPIVIGIIALLLLVYLRSLVAMVYLMATVLLSYFSALGAGWLILHTFFGTEAIQGLIPLYAFVFLVALGEDYN
IFMVSGIWKASKTHKLKEAIRLGVSSTSSVITSAGLILAGTFAVLSVLPIQVLVQFGVICAVGVLLDTFIVRPFLVPTIT
LLLGRAAFWPAKPGTPADERSAPRPGN
>lcl|NZ_CP026520.1_prot_WP_042230877.1_1917 [locus_tag=PC41400_RS09840] [protein=N-acetyltransferase] [protein_id=WP_042230877.1] [location=complement(2158137..2158958)] [gbkey=CDS]
MAQLPVLTKELVKRLEQAETAFWTSRIEAIGSREGNPMGVHTARFGHTSAVCTEEMPWPLFNTVRGLGPEDVGELGTILR
FYAKYGRSCTFELTPGKADKSVAFALARQGFYQESFHAYMYGLPEPIAPVLPPHIRIEPVTSADRFDLYAELHCVGTGMD
VSAKRHFIDNNIVLLNRPDWHLYMGYVNDQPAGVSMMHIDGDIASFTLAATLPQHRSRGLHTAMLQVRMHEAAKRGCRLA
AAQAAFGSVSQHNMERAGMRLAYTRAVWSALGE
>lcl|NZ_CP026520.1_prot_WP_129112467.1_1918 [locus_tag=PC41400_RS09845] [protein=nitroreductase family protein] [protein_id=WP_129112467.1] [location=complement(2159067..2159714)] [gbkey=CDS]
MNYEDFKSIVLDRRSVRKFTEDPVSVEDVRELIDCARYAPSDTNSQTWKFIAVMNGDLIKTIEDVTWEELHKRAADAEER
GLFREGRLLVKSFGPYATAFSGAPVLIICLATPYDSKFRKRIFDPIALVENSVWDEEGIKSSCLAAQNLMLAAHARGLAT
CPMTGPVLLAQHRLRELLPIPEEAQVNMVIALGHPQERPGKLARKEVDEILEIIR
>lcl|NZ_CP026520.1_prot_WP_042230874.1_1919 [locus_tag=PC41400_RS09850] [protein=efflux RND transporter periplasmic adaptor subunit] [protein_id=WP_042230874.1] [location=complement(2159858..2160955)] [gbkey=CDS]
MKKKLWALAAAVVVIGISAFLYIKSHPQTGPNPADMAMNNPLSFKVTREELVGSVEVKGKSSYEKETWVHAPFSGDIVKW
NVNEGAQVKKGDSLFVLDSAAIDAEIVQMQANLRKQDLEAQLAKFQERSAEAPSAGSGESGGMTESESKKRYVDAESAKL
QAQLNTMNREAAQSQLGEKQKMLAKANFKAPESGIFLFEDSSKIPQALQANHRIGKIVDLSKLELLCYVGEFDVFQIKPD
MPVQVKINALADVPLKGKVKQVSKFAKSGTEAVASPTAQFEVIVSLEPNERLIAGLSLTGTIETARKADALVVPTLAVQR
DKDKHFVMLETAGGVERKEITIGMETADKTEVVSGLNEGDTVVLQ
>lcl|NZ_CP026520.1_prot_WP_042230873.1_1920 [locus_tag=PC41400_RS09855] [protein=ABC transporter permease] [protein_id=WP_042230873.1] [location=complement(2160973..2162370)] [gbkey=CDS]
MKMRDYMGLGWDQLKRRKVVTALCALGISIGSAAIVVALAVGESASHYSEQQMKAYLKTDEITIRNDQSQDQGSAGSASD
KPGSAAGSITNQKLDLIRTLPHVKAVASFQQVGHFFFKLDDGKIGSFELVATDLDTLEKFGNEFQQGGPSDIENTIILNP
GATMGLLDEETMKNRDRQMRANQNDPAIWETFREFDKIPTSLYQKKLTLTLEHPSASEGKAQAQIPVRVAGVLKKGESVP
ANMVMYDKKGYISHALAKQIRETIKASAPPDSETAKRMGAGESYSEVLVKVENSSHIETVDKQIQKLRLNTQNNLHQEER
MQGEFVIIRVVFLGVGLFVLFIASISIVVAMTMSTYQRRRQIGIMKVLGANLRQIRNMFIVESALLGLLGGILGIMLSYW
VVWGINALTFMMGKNSAGDDSQILFISTWILPVGLFFAIMTGVLSGIYPAVKASKTDALTAIKRE
>lcl|NZ_CP026520.1_prot_WP_042230872.1_1921 [locus_tag=PC41400_RS09860] [protein=ABC transporter ATP-binding protein] [protein_id=WP_042230872.1] [location=complement(2162367..2163149)] [gbkey=CDS]
MLEVSGLSHTFLSGTEQSQVLKNINFTVNKGEIVALLGSSGSGKSTLLNLMAGLMKPTEGSILIDGQPIERMSENKLAEF
RRHHIGFIFQSYELISHLTVRENVELPLVFQSVKPSERRERAMSLLTQVGLAEKTELFPSQLSGGQQQRVSIARSLITRP
PVVFADEPTGNLDTKTEKEIIDLLLELNRTLQLTFIIVTHESEVAARTQRVIQLKDGRLAEDSDYLHGENETAAALADEA
PPALAPETALHAGQAGGERP
>lcl|NZ_CP026520.1_prot_WP_241688834.1_1922 [locus_tag=PC41400_RS09865] [protein=MFS transporter] [protein_id=WP_241688834.1] [location=complement(2163142..2164344)] [gbkey=CDS]
MVALKKQNDFVLRGFTFSYYMMMAVIATFFPLYFDSRGYSKAEIGLLYSIGPMVSIFANLAWGFASDRMQKIKPIMITLY
VGQLVFVISMFGTDSFTVLYALMAGFYFFQQPAQSLNDSQLLLHASQTGKSYSSFRIFGSIGFAFSALFFGLLIKETNIS
LTSSISIAVIVIGLIFTLGLVDTRVAYSKMDFKGMGKVIGSPRLLIFLLFVLIMSISHRINDGFLALHLRELGASDMTVG
TAMMLSSLSEIPVFFLLGKYGNRYKELPLLAIAGFMYAVRLLIVSLTDQTGWVLFAQTMHSVTFGIFFVTALRYLAQIIP
DEYRASGQAIFAVTWSGLAGIISGRFGGLIYADYGSTVLFGIGSGLALTAAAGFLVFHARLNALEAKLTAISTKKEDSHA
>lcl|NZ_CP026520.1_prot_WP_042230870.1_1923 [locus_tag=PC41400_RS09870] [protein=hypothetical protein] [protein_id=WP_042230870.1] [location=2164755..2165069] [gbkey=CDS]
MTCRFIFDERLGIPVPEFDKEWEEYSLRERESILFQWEQIRGTIPDRIKQLEGQIIAKQNKLNVEDDFPVSCALNSQIAE
LASAINDLHLWFRTNQEVESKIHG
>lcl|NZ_CP026520.1_prot_WP_042230869.1_1924 [locus_tag=PC41400_RS09875] [protein=hypothetical protein] [protein_id=WP_042230869.1] [location=2165246..2165875] [gbkey=CDS]
MPTENEKPSWIRLIVKPLLGRQPSLYPIEEWDSLIDVRIDKLQVSALSGDSPDIEPYALALKSGLYLLNESLDNSHEISQ
SVTNETGSYWHGLMHRMEGDYSNAKYWLQDAGNHPIHTDLVNEVRSFLDRQPGLRAIEHEAMKAKIDVLVTSPEWNSSVF
ADAVELQVTLLQNPDIEAWLTHIQYLEFKLLLAYCYRQCYGGELVELAE
>lcl|NZ_CP026520.1_prot_WP_129112559.1_1925 [locus_tag=PC41400_RS09880] [protein=tetraprenyl-beta-curcumene synthase family protein] [protein_id=WP_129112559.1] [location=2165913..2166938] [gbkey=CDS]
MYRVYKYVLPGVKRELEGWRREAERIPDEELRTQAIASMTSKQFHCQGGAIYAVINLPMRQVLISLIVAFQTISDYLDNL
CDRSTSLDPQDFRMLHQSMLDAVDPAAPLHDYYAYRNEKEDNGYLVKLVQTCQKSIAELPSYGLAADAVRSLVGLYCDLQ
VYKHIRPELREQALLDWWAEHSANYPDLTWNEFSAATGSTLGMFMLFAAASDRNLDRRRVEGIKSAYFPYVCGLHILLDY
LIDQEEDRVGGDLNFCFYYADMDETIRRIAYVAEQAKEGTSKLEDAAFHRMVIEGLLALYLSDPKVKDQPVVRQISRNLM
KDSTISRLFFWINSLVIRKTS
>lcl|NZ_CP026520.1_prot_WP_009675013.1_1926 [gene=pfkA] [locus_tag=PC41400_RS09885] [protein=6-phosphofructokinase] [protein_id=WP_009675013.1] [location=2166981..2167949] [gbkey=CDS]
MTAVKSIAVLTSGGDSQGMNAAVRAVVRSALYHGVKVFGVQRGYQGLINNDITEMDLRSVGDIIQRGGTILQTARSKEFT
TPEGQQLGAENLRNRGIDGLIVIGGDGSYQGANKLSKLGIKTMGLPGTIDNDIAFTDFTIGFDTAVSIVVDAINKLRDTM
TSHERSSVVEVMGRHCGEIALYAGLASGAETILVPEVPHDCQEIAKRMSDNFAHGKRHSIILVAEGAGKGDDIARQIMEH
NGIDARVTVLGHIQRGGTPTHTDRILASRLGDFAVRRLIEGDSGKSCGVIKGELVATDIDLVVNTKRPFNMELYNLALRL
SQ
>lcl|NZ_CP026520.1_prot_WP_042230868.1_1927 [locus_tag=PC41400_RS09890] [protein=D-2-hydroxyacid dehydrogenase] [protein_id=WP_042230868.1] [location=complement(2168116..2169072)] [gbkey=CDS]
MLNITALTKMDLTYLQNHLPEDLKDAVSLAQFPTAEEAGDRLASTDILLTSGAFQDDWLPLLPKLRWFQSLSAGVDKLPL
KKLDERGIQVTNMSGVHSIQMSEYALSVMLDFVRRSHTLMESQSLKEWNHRIRTAELHGQTLGLIGTGAIGSEVARKAKA
FDMKVVGYSRSGRDREHFDDMRKGKSGLYSLLRESDFVVVIVPATRETENLIGAEELALMKPSAFLVNMARGSVIDETAL
IEALRKGILAGAALDVFREEPLPADSPLWGMEQAALTPHIAGNTPKYVERCAEILFPNIRRFIENKPLINRIDPEHGY
>lcl|NZ_CP026520.1_prot_WP_084706622.1_1928 [locus_tag=PC41400_RS09895] [protein=DUF3906 family protein] [protein_id=WP_084706622.1] [location=2169227..2169451] [gbkey=CDS]
MKVFLYKVELELKESLAYLIVVAETDAKAFDAVEGHLVRHFTVAPEVVQTTIIEKKRVGNGAGYVIESAVESPV
>lcl|NZ_CP026520.1_prot_WP_042230867.1_1929 [locus_tag=PC41400_RS09900] [protein=NAD(P)H-dependent oxidoreductase] [protein_id=WP_042230867.1] [location=2169471..2170013] [gbkey=CDS]
MKIVIISGSNREGASTTLLSAYVGQVLTEKGAEVTLVDLYKQPVPLYSPDPSVVEDENLVHMKQAVMEADGIILATPEYH
GAMSGVLKNALDHMDAQYVDGKVVLSMSSAGGAVGVSSLQQLQTVVRNLHGVNCPEWISIGGAGRTFNEQGEPETLQIRE
RILRTVTYFLSMTETFRKSV
>lcl|NZ_CP026520.1_prot_WP_042230866.1_1930 [locus_tag=PC41400_RS09905] [protein=aldo/keto reductase] [protein_id=WP_042230866.1] [location=2170040..2170936] [gbkey=CDS]
MEKRILGRTGLQVSSLGFGGAEIGYQGIDPGTVENLLGRALDAGLTVIDTGECYKDSEELIGKAVSHRRAEFSLFTKIGH
SAGFQEEDWDLGMLEKSIDRSLKRLKTDYVDLLQLHSCPEEILRRGEVIGLLEKFKKQGKTRFIGCSGDGQAALYAVRTG
LFDTLQTSLSIADQEAIELTIPEARQRGMGVIAKRPLANAAWRTGEKPADSYHHEYWERLKRLRYHFLNESVPEAIAYAL
RFTLSVPGVHTAIVGTTRPGRWEENAAFLSRGPLPIEAIAEIRGRWQQSSRGNWPGLR
>lcl|NZ_CP026520.1_prot_WP_042230865.1_1931 [gene=tatA] [locus_tag=PC41400_RS09910] [protein=twin-arginine translocase TatA/TatE family subunit] [protein_id=WP_042230865.1] [location=complement(2171098..2171325)] [gbkey=CDS]
MNFFSPWHLLLVVLVALLLFGPSKLPELGRGFGRMFREFKDATNGLVSDEPNQNQAAPARKEVNTNSSVEEEARK
>lcl|NZ_CP026520.1_prot_WP_042230864.1_1932 [locus_tag=PC41400_RS09915] [protein=hypothetical protein] [protein_id=WP_042230864.1] [location=complement(2171458..2171652)] [gbkey=CDS]
MEMETVKLSHIVKTYTPELYTFLTRTELNSEIVLVNGMQSLDQEDVIEIVQFSINENRKDAFLQ
>lcl|NZ_CP026520.1_prot_WP_042230863.1_1933 [locus_tag=PC41400_RS09920] [protein=YitT family protein] [protein_id=WP_042230863.1] [location=complement(2171803..2172624)] [gbkey=CDS]
MDKKAISIAVVLIGAALVAFGFNMFLIPHELLSGGVSGIAMVIGYLTNMDISILYFVLNLPLIIWGLVIIGRRFIVLSLA
SVIATTWFMQLIPVRGLVQEPILGAVFGGVLVGIGAGITIRAGGSTGGFDIVGFILTRKRDFPLGQMLFALNGGVIFMLG
YFKNNWDLALFSMLSIYIAGKVIDTIHIRHIKVTAFIVTKQREALLEKLLVIPRGVTVIRTEGAYTNQEQSLLMTVTTRY
ELVGLQKIIKETDPKAFVNVVETVGVLGEFRRQ
>lcl|NZ_CP026520.1_prot_WP_042230861.1_1934 [locus_tag=PC41400_RS09925] [protein=DUF2188 domain-containing protein] [protein_id=WP_042230861.1] [location=complement(2172783..2173199)] [gbkey=CDS]
MPWTKNDYPPSMKNLPERVRKKAVEIANALLDDGYEEGRAIAIATAKAEEWEDNHPDHRSGGKSGTERRASSSHSSPLHV
VPDGDGWAVKAEGEDEPRYEADTKSDAVKKADELASEANTRAVIHRKDGTVQDSHNYS
>lcl|NZ_CP026520.1_prot_WP_042230860.1_1935 [locus_tag=PC41400_RS09930] [protein=DUF4247 domain-containing protein] [protein_id=WP_042230860.1] [location=complement(2173395..2174102)] [gbkey=CDS]
MNRWTSWISCLLVFVLLTGCAPNAGDFVRQEYNLTSADGSGSNLSKVYTAENKDVPTVAKELAAREKPREISKESQDQMF
LLYPDKVINLQRDEQNPANTLVQLDSIQYAREHYSPSFLQTFLTAAMLQSVLGGGFFGNRGASDYRGYRSTPTYRGTPGG
YSPGTGTGTTKPPVTSDREGSFRTPGSPSGTTGGIGAGTGAGGRRSEITNPGRTKPGIGTRPGTSGRSGSFKRRR
>lcl|NZ_CP026520.1_prot_WP_042230859.1_1936 [locus_tag=PC41400_RS09935] [protein=DUF4178 domain-containing protein] [protein_id=WP_042230859.1] [location=complement(2174253..2174786)] [gbkey=CDS]
MSIFKRVGNIIRSNKQQEPVRQANPIEESRVGDIVNVDLEEYVVTGKVIYFDRGFEPHRYAYYLQSGKDIRCLLIEKGRS
YECFLCDFMEGALDDPNDVPSRLELDGEDTFELEHHRSDVTRTEGNTDFRSGDEVTIWRYFGPDDRYFFLQWQDGKFVAM
QGERTPATQVKFMRATL
>lcl|NZ_CP026520.1_prot_WP_009675007.1_1937 [locus_tag=PC41400_RS09940] [protein=DEAD/DEAH box helicase] [protein_id=WP_009675007.1] [location=2175088..2176695] [gbkey=CDS]
MKNFSEFDLDAKVIKAITEMGFEESTPIQEKTIPLAIEGHDLIGQAQTGTGKTAAFGIPLVSKIEPSEDKIVALIMCPTR
ELAIQVAEEVSKLGRFKGIRSLPIYGGQDIGKQIRALKKRPQIIIGTPGRLLDHINRKTIKLEDVKTVVLDEADEMLDMG
FMDDIQSILSLVPAERHTMLFSATMPPNIQKLAQQFLTNPQHVSVIPKHVSAPLIDQAYVEVHEKQKFEALSRLIDMESP
ELAIVFGRTKRRVDELAEALQKRGYAAEGLHGDLSQNQRDNVMRKFRDGSIEVLVATDVAARGLDVSGVTHVVNFDLPQD
PESYVHRIGRTGRAGKEGTAWTFVTPREIDHLHFIEKVTRHRIPKKPMPSPAEAMEGKQKVTAERIINLLQEDKHNEFKG
VAIQLLEEYDSVNLLAAALQLLTGGEKKEVSVELTPEDPIRAKKRRVDVRSSGRKFNYGSDRRGDGRSGGGYGSRSGQGG
SRYGSSSSRGRDGGRDYRDRDRDSRSGGGYGSRSGGSDYNRDRSSSTRRSEDSNA
>lcl|NZ_CP026520.1_prot_WP_084706621.1_1938 [locus_tag=PC41400_RS09945] [protein=sporulation protein YjcZ] [protein_id=WP_084706621.1] [location=complement(2176783..2176875)] [gbkey=CDS]
MSGVGGGFTSTGAILVLFILLVIISRTFLY
>lcl|NZ_CP026520.1_prot_WP_042230858.1_1939 [locus_tag=PC41400_RS09950] [protein=DUF624 domain-containing protein] [protein_id=WP_042230858.1] [location=2177015..2177821] [gbkey=CDS]
MEFRGLMGGFYRISEWIMRLSAINLLWILCSIPFFFFALAGLTSGQTDIMMQSLVLMGILAPFTLFPSTAAMFTVARKWV
TGDEDVPLFKTYFRGYKENFLQSMIGGIAFVLFILILFVSYRFYAGLDSALSLLSYMFLAMGILLIGAFFNFFCIMTHLH
MKVLQIIKNAILLTIGNPVTSVMTIIGNVAIVMISFRFSFLLVFFAGSVSATFTFWQFNRSFEKIKRKQEAMEEAERERE
EEENGQAAELDAAAVREDDSTKKESDKL
>lcl|NZ_CP026520.1_prot_WP_042230857.1_1940 [gene=fsa] [locus_tag=PC41400_RS09960] [protein=fructose-6-phosphate aldolase] [protein_id=WP_042230857.1] [location=2178317..2178973] [gbkey=CDS]
MKLFLDTANVNEIKEAHELGVISGITTNPSLIAKEGRDFFETLKEIVGIVGDLPISAEVVATDVEGMVQQGEKLAKIGSG
IVIKVPMTADGLKATKRFSDQGIKTNVTLVFSSTQALLAARAGATYVSPFIGRLDDINQVGMNLIEEVSQIISVHELPTQ
IIAASVRHSTHVIEAALLGSHIATVPFKVIEAMMKHPLTDAGVERFMKDWEGANQKTF
>lcl|NZ_CP026520.1_prot_WP_042230856.1_1941 [locus_tag=PC41400_RS09965] [protein=DUF1499 domain-containing protein] [protein_id=WP_042230856.1] [location=2179211..2179621] [gbkey=CDS]
MLHEGGTTLKRTLVGLLRSFETTGEKAKDPELKTRYYKLSKDQVWEETVGMMKKTPGYKLLHEVKNVGEIVVEKRTALGR
TQDITITLFAVNPTTTAIDIYSASRGNMGDLGSNYRTILDIYKQLDKKLAAYKISH
>lcl|NZ_CP026520.1_prot_WP_016423958.1_1942 [gene=tpx] [locus_tag=PC41400_RS09970] [protein=thiol peroxidase] [protein_id=WP_016423958.1] [location=complement(2179723..2180241)] [gbkey=CDS]
MSQERKGAATFKGNPITLVGPELKAGDAAPDFKLNKDLLNEVSLSDYAGKVKLISVVPSIDTGVCDAQTRRFNEEAAKLG
DNVAVLTVSVDLPFAQARWCGAAGIDQVVLLSDYKAKSFGQAYGVLIQELQLDMRSIFVIDANDKIQYVEYLGEMTEHPN
YEAAIEAVKKLV
>lcl|NZ_CP026520.1_prot_WP_042230855.1_1943 [locus_tag=PC41400_RS09975] [protein=LysR family transcriptional regulator] [protein_id=WP_042230855.1] [location=2180424..2181347] [gbkey=CDS]
MELRQLQYFVKVAKKQHMTQAAEELRVAQSAVSRQIRHLEEELGVRLFVQKGRNLQLTPAGSLFLSRAEHILTDLERAVG
EVREFLDPEAGEIRVGFPHSVSVSILPAFISCFKKEHPNVKFRLKQGTYTSLIRDVTEGEIDLAIISPCPETHELVKGEV
MLTEELLAVLPPNHALAGCKSIRLEQLKEEPFVMFGEQYSLRPIVMNACLKAGFVPKIEFEGDETDTIRGLVAAGMGVSL
LPELALRYSTHIEPCVVKVDPKVTRTVGIIRRRDEKLPPVTELFGRFLMEFQPSADPGRPMDRLDGA
>lcl|NZ_CP026520.1_prot_WP_009674804.1_1944 [locus_tag=PC41400_RS09980] [protein=zinc metallopeptidase] [protein_id=WP_009674804.1] [location=complement(2181470..2182156)] [gbkey=CDS]
MFFHPMDFLILIAFGLSMWASFRVRGTFNRWSQVPVQSGMTGYEAARRMLDAEGLYDVPIEPVPGALTDHYDPISRTVRL
SEPVYYESTISAVSVACHEVGHAIQHQVHYPMLVARHKMFPVVNFASGVAPLLLIAGLIFQQFDILLLLGIVFFSAAVLF
QLVTLPVEFNASNRARELMIAGGFITNEEEQGVAKVLNAAALTYVAAALISLLELVKYVMIFNSRSND
>lcl|NZ_CP026520.1_prot_WP_009674805.1_1945 [locus_tag=PC41400_RS09985] [protein=MerR family transcriptional regulator] [protein_id=WP_009674805.1] [location=complement(2182197..2182628)] [gbkey=CDS]
MKLHRIGELSRLAGVSPRTIDYYTKLGLIHPEKRSDTNYRLYSDETLLRLKRIEEMKKEKYTLEEIRTTMLKLDKAAHDE
HVTDKLTSYQVLLGQLQKEIKELDPMIQQLKPNQLKKIYNLITQPTAACVEALLYLLGKGPFT
>lcl|NZ_CP026520.1_prot_WP_042230854.1_1946 [locus_tag=PC41400_RS09990] [protein=ammonium transporter] [protein_id=WP_042230854.1] [location=2182936..2184363] [gbkey=CDS]
MIKKTLIGLALLSMALPTLAFAEAPTPEQLQAAINSVWVLVAAMLVVFMQAGFALLEAGSTRMKNAGHVAGKTILTFGIC
AIAFWAFGFGLAFGDGAGLFNNNLFGLSGFFMDGVNADAFVSLSASDVPITIKFLFQLAFAGVSLAIAWGGFAERAKLPV
YFIFGLLFTVVIYPIVAHWVWGGGWLSKIGMQDFAGSTVVHLQGATAALVATILLKPRIGKFNKDGSPNKIPGHNQVLSV
LGVIIIWIGWFGFNPGSTLSAMGDGFFGYIALTTNLAAAAGGVAAMAVSWIYFGKSDIPSMLNGVLAALVAITASCAFVT
PLASIAIGAIAGGVTFFTAIWFEKKGIDDPVYAFSVHGIAGIWGTLSTGFFATPELAKTVNVGGAGLFYGGGFRQLWVQF
IGVAGALVFVLVISFIVLYLIKITTGLRVTEEEEIMGLDLAEHGTYGYPEQMNHAIEAHTSAKQSNEGGRPVPVS
>lcl|NZ_CP026520.1_prot_WP_042230853.1_1947 [locus_tag=PC41400_RS09995] [protein=DUF294 nucleotidyltransferase-like domain-containing protein] [protein_id=WP_042230853.1] [location=2184402..2185529] [gbkey=CDS]
MKPTLGSDWQLLRTRIARAAGTDELALIRADMQDLFRKELGEKAAALLYKELNFTHDTLIRKMILLAEKRLAAEGAGEPP
LPYAFLLMGSGGREEQTPWSDQDNGLVYENPPEGTEAAADVYFSKLAQRIRQGLELAGYPPCTGRVLAENSMWRRSEAGW
KDQLSGWFEEPVWENARYLLIASDLRCVYGSEALAERIKQAFHESVALNRGLLLPLLANTLHHKVSLGLLGRLVTERYGP
YAGGVDIKYGLYIPLVNAVRLLALENGISDSSTFKRIHALIDRGAVTVEFGTACREVFLHSLEMRLSTPFYRENGHFATA
GILPAEMLTRKCRAELKNGLKFGKRLQRQIIRQVRGSVKKGRDASSAKLREHTGQ
>lcl|NZ_CP026520.1_prot_WP_042230852.1_1948 [locus_tag=PC41400_RS10000] [protein=exonuclease domain-containing protein] [protein_id=WP_042230852.1] [location=2185550..2186278] [gbkey=CDS]
MKESGPGGGLWRLYKMGGLTPAITSMFNPRNAQQMAFIRQLLKDQRRHSLLDTSLHNLKVVVFDLETTGFSSAADEIISI
GAVSMAGDKLEEESFYTLVNPNRPIPPHIEELTGIRTEEAAEAPDLITALREFFEFVGARVLLAHGSAHDKAFLNAALWK
TSKISLSHRVLDSLMAAKWLMPRHKDYTLDSLLELYGIPVLRRHHALDDSLMTAELWKKLLEEIGEMGVDTLGDLYALLG
KA
>lcl|NZ_CP026520.1_prot_WP_241688835.1_1949 [locus_tag=PC41400_RS10005] [protein=Mov34/MPN/PAD-1 family protein] [protein_id=WP_241688835.1] [location=complement(2186366..2186986)] [gbkey=CDS]
MYTEWLARFSSFPDEETCGFVYGRIRNETAYVSSFVWVPNVSACPARSFRISPSDIMRHLGGGQARNDRPARSSLAAPVL
LGLFHSHPRTAASPSSEDISALRLWPELPSFWIVSLERCVPETEVYAWAEPQDSPAALGPSESADSADKIVTTEDTGQPG
SVGPSTTDTLTDRTSLRAVSAPSSNRPNQAKSLSKLRKIPLCIEAL
>lcl|NZ_CP026520.1_prot_WP_042230850.1_1950 [locus_tag=PC41400_RS10010] [protein=MFS transporter] [protein_id=WP_042230850.1] [location=complement(2187071..2188375)] [gbkey=CDS]
MEKTKSRGLDIIFESDQDDDYQFTAEQTAEGGEKGGGGKGRLLEFMAIATVPLVLVLGNSMLVPILPQLETELGITKFQS
SLVITLFSITAGLIIPISGYLSDRFTRKSVIIPSLIIYGAAGVLAAFGAIWKSYTILIIARAIQGIGAAGTAPIAMALVG
DLYKGGTESKALGLTEASNGLGKVLSPIIGSLLALLVWYMPFFAFPIFCLLALLAMMFLIHEPKTKKKPTELKPYLHKIG
TILKKKGRWLITSFFAGSLALFILFGVLFYLSNILEEPPYHIDGVKKGFVLAIPLLGMVITSYTTGALIKKNGTLMRWLI
NIGLLVMTLSLASTIFAFQNLYLFIGLLTLSAIGTGLLLPCLNTMITGSVEKSERGMITSIYNSLRFIGVAFGPPIFGWL
MDKSDRLIFITVSSLAGLTLILVFYLVKPQGKIS
>lcl|NZ_CP026520.1_prot_WP_042230849.1_1951 [locus_tag=PC41400_RS10015] [protein=TlpA disulfide reductase family protein] [protein_id=WP_042230849.1] [location=2188554..2189090] [gbkey=CDS]
MKRNAAILAIVLLLAAAIFVQIGKKDEKQTAAPTQTGPRPGMFVPSFSLPDLDGKPYQVGGARDKPLILNFWASWCGPCK
MEVPDLIDLQAKYGDKVDLVAINVTSQDDEQKARAFAKEYGINFPVLLDEPGKVSESFKFLAIPTSFLVDKNGVVQEVVN
LLPKAEWEKKIKALIKGR
>lcl|NZ_CP026520.1_prot_WP_042230848.1_1952 [gene=cimA] [locus_tag=PC41400_RS10020] [protein=citramalate synthase] [protein_id=WP_042230848.1] [location=complement(2189237..2190853)] [gbkey=CDS]
MSDSIKIFDTTLRDGTQGEGISLSVEDKLKIAQKLDELGVHYIEGGWPGSNGKDIEFFNRVQELNLKHAKVTAFGSTRRK
NTKAEEDPNLTTLVASGVKVATIFGKSWDFHVHTAIQTTLEENLAMIFDSVRYLKSCGLEVIYDGEHFFDGYKNNPDYAL
QTIRQAQEAGADWIVLCDTNGGSMPEEIFEIVSEVKKQITTPVGIHAHNDCELGVANSLAAVRAGARQVQGTMNGYGERC
GNANLCSVIPNLQLKLGFACISDEQLESLGPVARYVSEIANMHMPMNQPYVGSAAFAHKGGIHVSAILKHASTYEHIVPE
KVGNKQRVLVSELAGQSNLVSKAAELGLNLEQQDGKAKEIISRIKDLEHEGYQFEGADASLELMMREASGELQEPFVLES
FKIFVEKSADRPVSSEAFVKVNVNGESMYTAAEGDGPVNALDNALRKALLPFYPDIHRMHLSDYKVRVLDDKGATASKVR
VLIESRNFENEWNTVGVSSNIIEASWHALVDSFRYALIGKSSIERESLDPRPVGLVNH
>lcl|NZ_CP026520.1_prot_WP_042231180.1_1953 [locus_tag=PC41400_RS10025] [protein=DNA polymerase IV] [protein_id=WP_042231180.1] [location=2191112..2192392] [gbkey=CDS]
MAQEGYPKNGRVILHIDMNAFYCSVHEVVQPELYRGKPIAVSGSVEQRKGIIVTSSYPARARGVKTGMLVRQAMRLCPEL
VLIKPNFDLYRQFSRGFRRIVYDYSPMVETMSIDECYVDITGSKQFGTPLQIAEEIQTRIREELGLPCSIGVAPNKLLAK
MASDMKKPNGLFVLRLRDVPKVLWDRPCGELYGIGRKTAEKLQKLQIHTIGQLAAADEGLLRKSFGINGFALKQAANGIS
RSEVNPEREPNKSVGHTTTLPYNYTDRKEVFRVFLNLADQVGRRLRHQKLMASTVQITIRDPDMKTITRNVTLPVPTDHH
EDLYKAACELFERHWEPEKPIRLLGITGQNLTPKEEAAIQLDLFSYEQQPKKDKLSEIMDALRDKFGEDAVLTAGMIGDN
PSAHIRNRKRRGTSLQKDPDGDSFPL
>lcl|NZ_CP026520.1_prot_WP_042230847.1_1954 [locus_tag=PC41400_RS10030] [protein=ferredoxin] [protein_id=WP_042230847.1] [location=2192551..2192787] [gbkey=CDS]
MAKYTWVEKDTCIACGACGATAPDIYDYDDEGIAEVIYENDGNHGVTVIPEDLYDDLQDAQDGCPTDSIKVADTPFNM
>lcl|NZ_CP026520.1_prot_WP_164977117.1_1955 [locus_tag=PC41400_RS10035] [protein=L,D-transpeptidase] [protein_id=WP_164977117.1] [location=2192998..2194500] [gbkey=CDS]
MSTEKPDEKLAQFLRHNPIEDRVYLKEYVRDHPDQRMAWYLLGREYAARGDRGKAAYCYAQSGEIYEAFEKRKITVDWSA
LDSSVPWPQELKTAAKRSRRRKAALTAALLALIALLPAPAGPAAGDGRPAGSAAASAELPAADGGLRVYLAAEKQTSASG
ALERMLLRDRPTGGEAVLLSAQSADGPGGGWIAWHRPLNPLLSVASGGPGEAAAIAYFDAAACNCRPDDPGRFAEPAGAW
RQRQEQLLVLRSALEAYRAASGKLPEGPEELLRPYPDNVMSGLTPYMREAFPLVREAAAAGANGQAAAGTGGAGTAPGAK
PTAASVAAEEALSEPLAILIDTKLHRLALVSGRTVVRSYEVGLGGARTPLGEFAVSEKVRDPKGRKKGEFGTRGMTLSDT
DYAIHGTRHPSSIGKDESLGCIRMRQEDVEELFDMAPKGTKVTIGSGLLPAGLVGDGSGGDSGASKGSGAPGAGFGPKGA
FKLPSEVKETNPGKVYRWLD
>lcl|NZ_CP026520.1_prot_WP_042230846.1_1956 [locus_tag=PC41400_RS10040] [protein=quinone-dependent dihydroorotate dehydrogenase] [protein_id=WP_042230846.1] [location=2194897..2196000] [gbkey=CDS]
MVYKKIAKPVLFRMDAEKAHHLIINGLKTAAKIPGMSSVIGAAWGSGHAPELHSTLWGIAFPNPVGLAAGLDKNAQAVEG
FSRMGFGFMEVGTVTPKPQSGNDKPRLFRLPEDEALINRMGFNNVGAEAMAQNLSRLGSRPVPVAVNIGKNKTTPNEQAE
EDYRSCIRVLYGHGDFFVVNISSPNTPDLRNLQHGNDLQRLLHAVTGEMKIQHERTGGAAKPVLVKIAPDLTDTELENTI
AVIGGSGVSGIIVSNTTLGRDGLSHPNRDQAGGLSGRPLTSRSTELISRVYRMTEGRIPIIGSGGIFTAQDAYDKIRAGA
SLVEVYTALIYEGPGLLKKLNEGLSTLLKKDGFTHISEAIGADQRRL
>lcl|NZ_CP026520.1_prot_WP_009674739.1_1957 [locus_tag=PC41400_RS10045] [protein=GTP pyrophosphokinase family protein] [protein_id=WP_009674739.1] [location=2196022..2196825] [gbkey=CDS]
MDGRDWKKFLFPYEQTVEELKIKFKTLRAELKNREEYSPIEFVTGRVKKISSILDKAKKLNVSMDDLETGIEDIAGIRIM
CQFVEDIETLRELIHQRKDMKVLYEKDYIANKKESGYRSFHMIIEYPVQTSLGMKKVLGEIQLRTLAMNFWATIEHSLNY
KYKESMLPEEVRTRLSRAAEAAYLLDQEMSNIRGEIVEAQKLFEDNSNLVTVILNCIQVLYFYHRVREAVQFQMRFNEIF
EKDDTRAMRELQEEIQEAIKRAKKGAR
>lcl|NZ_CP026520.1_prot_WP_042230845.1_1958 [locus_tag=PC41400_RS10050] [protein=DUF309 domain-containing protein] [protein_id=WP_042230845.1] [location=2196827..2197234] [gbkey=CDS]
MRPYDPLYLAFLYEFNVNRDYYECHEVMETLWLEEGRKPFYQGLLQIAVGLYHHLNANVSGAVKLMESGIAKLQGYPDRL
LGIDVALLLRDAEAYLGKLRNNGVRPFEPYPLEIRIVDEDLAERVRQIQGERESG
>lcl|NZ_CP026520.1_prot_WP_042230844.1_1959 [locus_tag=PC41400_RS10055] [protein=GNAT family N-acetyltransferase] [protein_id=WP_042230844.1] [location=2197411..2197893] [gbkey=CDS]
MADYQSRPLSPDDLDVICGMPQHAGELFYMYPKGVFPLTPEQIREALKTRLCPTVITAEGGEIAAYANLYDCVREKECWL
GNVIVALGYRGKGASSYLIRTMMDKAKRELGVPFLNLICHNTNTPALLLYTKIGFCPFDVSEKAGPDGKRLAGIHMRVSL
>lcl|NZ_CP026520.1_prot_WP_042230843.1_1960 [locus_tag=PC41400_RS10060] [protein=DMT family transporter] [protein_id=WP_042230843.1] [location=complement(2198127..2199071)] [gbkey=CDS]
MKAAPLTKAYLAAVLYALIIGFSFIFVKTALVSTGPFDLLAHRFTISLAALLVFLALGRTRLNIRKRDYLAILPMAVLYP
VLFFAFQTFGLVQASASEAGIIQAAAPVFTMMLASVVLKERTGLRQNLFIALSVAGGVYIFAMKGLDFRTMNGAGLFLIL
LSALSSAGYSVLARKLSRNYTPLDLTFVSLLFGFIAFNGFAVIRRTVQGTLPLYFEPFGDLSFTVSVLFLGVLSSLASSF
LSNYALSKMEAFKISIFSQLATIVTMAAGVIILHEKLAYFHLIGAALMIAGVVGVCLPASSRKISRQPERRLDA
>lcl|NZ_CP026520.1_prot_WP_042230842.1_1961 [locus_tag=PC41400_RS10065] [protein=PLP-dependent aminotransferase family protein] [protein_id=WP_042230842.1] [location=2199209..2200567] [gbkey=CDS]
MHKYEGILTDLERQIADRRYKEGSKLPSIRQLSESYGVNKSTVIRALEELESRHLVYSIAKSGYYVVQSSTDTAPDSPDV
IDFAVSAPDPAFFPYLDFQHCMNKAIDTYKNDLFVYGTPQGLPALLVMMQKLLADYQVFTRVGSLFVVSGVQQALAILAA
LPFPNGKTKVLIEQPGYHLFIEQLVTHGVPVMGIRRSREGIDLDELERIFRTEPIKFFYTIPRFHAPLGSSYPTDVKKKI
AALARKYDVYVVEDDYLADLEPDGKADPVHAYDEGGRVVYLKSFSKIMFPGLRIGIAVIPGSLVDLFSQYKRIADIDSSM
ISQGALEIYLKSGMFERHKHKIKAAYATRSLLFRQMLAEYGPLAGGAVQEAAGSEELSPVHTHLVLSPSVSQTELARRLR
KKSVVIEPIDKHYLPGFPRDNYLKLNVSQVGDEQIRRGIRLLMEELGRYGDR
>lcl|NZ_CP026520.1_prot_WP_042230841.1_1962 [locus_tag=PC41400_RS10070] [protein=GntR family transcriptional regulator] [protein_id=WP_042230841.1] [location=2200720..2201106] [gbkey=CDS]
MNIILSQSSGEPIYEQIRQQIQRQILQGELLPGEGLPSIRQLAKDLQISVITTKRAYEELEKEGLIDSVVGKGSFVSGLN
REYIREQQLKQFEAKLSEAVREARQLGITVDETIETVKALYEEEEEIR
>lcl|NZ_CP026520.1_prot_WP_042230840.1_1963 [locus_tag=PC41400_RS10075] [protein=ABC transporter ATP-binding protein] [protein_id=WP_042230840.1] [location=2201103..2201978] [gbkey=CDS]
MNAVEFKHISKAYSNFSIQDLSFAVPKGYITGLIGPNGAGKSTLIKILLHLAIPEQGRVFIEGMEMPIREKEIKERIGFV
SDESHFYEHLTVRQTGRLLASFYRNWNETQFQTYLERFELPAGKKISDLSKGMKMKFGLAAALSHGAKLLVMDEPTAGLD
PVFRRELLDLLADLMQDEQHTILFSTHITTDLDRVADYVAFMNRGKLVFSESKDNIQERYAIVKGGTELLDADIRKQFVS
VRETSVGFEGLVYNRPQAEKLFGGYAVIEPANLEDIMVYTVKGGYGHGGTA
>lcl|NZ_CP026520.1_prot_WP_042230839.1_1964 [locus_tag=PC41400_RS10080] [protein=ABC-2 transporter permease] [protein_id=WP_042230839.1] [location=2201962..2202585] [gbkey=CDS]
MVALLSKDFRVMRKELLWMAVYVVFFAITFSNSRMGGVVVQLVPSIMLLFISANHDVKNRNFILLGSLPVRRSDIVRARY
VSVLLFGVLGAGIGFIVSTVTSMIQGTGGASSFSWGASGLLGAMLLVYAAIYLPLHYWLGDKYLQVISVIMMILLFASTS
MLGFIVGATDKWGLPSPAAVAVIALTGVSVFFVSYLISLGIMKKKDF
>lcl|NZ_CP026520.1_prot_WP_042230838.1_1965 [locus_tag=PC41400_RS10085] [protein=thioredoxin family protein] [protein_id=WP_042230838.1] [location=complement(2202652..2203161)] [gbkey=CDS]
MKKLAIYLGIIVVLFVAIFLVTKSSNKGNDDAANRLYGVSASKLKPATQKLLEDPNYQNLIKPEELDKRIANKESFFLYY
YGADCPHCKVTTPVLVPLQKELGIDVKQFNLLEFQDGWKKYNIEFTPTLVYYKDGVEVERMVGGVPENGGTEGNTPAKFK
EFFLKHNGK
>lcl|NZ_CP026520.1_prot_WP_042230837.1_1966 [locus_tag=PC41400_RS10090] [protein=pseudouridine synthase] [protein_id=WP_042230837.1] [location=2203552..2204307] [gbkey=CDS]
MLAHLGYGSRSDLKKAVKNGLIEVNGIRVKDSGMQVDPEQDEITVSGEKIVYREFVYLLLNKPQGVVSATEDNRDRTVVD
LLSEAYAPFDVFPVGRLDKDTEGLLLLTNDGKLAHNLLSPRKHVSKTYYAKVEGAVGENDKQAFEAGVELDDGYVTMPAQ
LRVLTCDIIVDPPRSEIELTIQEGKFHQVKRMFQAVGKRVVYLKRISMGPLQLDPALGLGEYRELTREELELLQGKPSHD
SGPGVDDDLHK
>lcl|NZ_CP026520.1_prot_WP_042230836.1_1967 [locus_tag=PC41400_RS10095] [protein=Cof-type HAD-IIB family hydrolase] [protein_id=WP_042230836.1] [location=2204396..2205211] [gbkey=CDS]
MTYKLIALDVDGTLLNDDYLLTERTRETVRRVHEQGSSIVLCTGRGPASAIPVLEELGLEGTIITHNGGATVQSPGLELI
HQYAFRVSQIAPMIAYCREHEVHFDVCAPFDMYLERLTEIEQAMYKKFFVTPHLVADVTALTVPLVKFTLYSADPAVMDT
VERDWSLSRLYGDLRMIRSGDHFIDVMHADATKGNALRSLCESLDVRPDEVLAIGNYFNDLEMISFAGLGIAMANSPAGV
LEQADQITASNNEDGVAKALETHVLQTVRRD
>lcl|NZ_CP026520.1_prot_WP_042230835.1_1968 [locus_tag=PC41400_RS10100] [protein=MDR family MFS transporter] [protein_id=WP_042230835.1] [location=2205269..2206771] [gbkey=CDS]
MARSGNTDKTKVLLAIMVVTFLSSIEGTIVSTAIPRIVGDLGGLDLMSWVISIYLLTTAVSTPIFGKLSDLYGRKKIFMI
GTGLFLIGSTLCGIAQSMEMMIFFRAVQGIGAGAILPVTMIIVGDIYSFEERAKIQGWISGIWGLSGILGPLAGGLMVDY
VSWRWIFYFNIPFGIISMLLISAFLHENLEKTKRPIDYPGIVTFTISMTALIYALLSGGVQYAWGSPVILGLLLVSVVFL
FIFIAVEKRSPEPMLPLDLFKDRLLTLINSSGFILSAILVAVTIYLPLWVQSVNGGGATGSGFALIPLSICWTIGAALAG
RGFVRLGVRSAVLIGVGGITAGSVALALMTAATPGWALEVYTALFGIGFGFAFTAYTITMQSAVGWKLRGAAMSSHTFIR
TLGQTLGIAVFGMLYNTSLHTYQQAHPDQVGGIDLNKALDSHGRSAFSPETMAGIRELFAFSLSRIFWSLLVLAAIGIVL
TLLLPKRFKVEEEPEQAVEH
>lcl|NZ_CP026520.1_prot_WP_042230834.1_1969 [locus_tag=PC41400_RS10105] [protein=hypothetical protein] [protein_id=WP_042230834.1] [location=2207016..2208053] [gbkey=CDS]
MAQLVLRPDLKTAGGEVTEILVDGRFAGIISLVYREGDRMTGGIQLEDAVLSPAGKREVDAFVRNHIQWLVDAMNIRECD
VVLTYGSYDHIIGTPLLSDESQAVDYADDETTAEWVGDESVYGDYEEDELDDISMEEDAYDAEPDYELVLLEDSRGRVEY
QLFDPGSRLVAEASCRIYGSEAVCMLDWVFDPLEEEMELASNLIVADFQEEETENFLFHSRFDGEIVDSVEISLEEADEL
SELDSGEDEFSVILARDDGDTLTYEIYEQEYGGIPIGTATIDVSEDQLTGFIDFRQTDDSRQRQEIARLLMEELDKERDY
DSLHLSMLYRNKLIDELLFENVTIQ
>lcl|NZ_CP026520.1_prot_WP_042230833.1_1970 [gene=trxA] [locus_tag=PC41400_RS10110] [protein=thioredoxin] [protein_id=WP_042230833.1] [location=complement(2208190..2208507)] [gbkey=CDS]
MSLLNVSDQTFQETLDKNNLVLVDFWAPWCGPCKMQSPILDQLTEEAGEAVTIAKINVDDNPETASKFGVMSIPTLKLFK
NGSEVKTMVGVQSLQTLKELVNQNQ
>lcl|NZ_CP026520.1_prot_WP_042230832.1_1971 [locus_tag=PC41400_RS10115] [protein=nitroreductase family protein] [protein_id=WP_042230832.1] [location=complement(2208548..2209180)] [gbkey=CDS]
MTQQIFPETLKVIAERHSVKSYDPDFIFPEEHKELILNAANQAPSSWNLQHWKFLVIDDPSAKQRLLPIASNQKQVTESS
FVVAVLGDLQANLNAETIYGAAVESGFLTPEIKDTLVSQINGAYASGGSIPRDEAIRNASFASMQLMLAAKALGYDTCPM
GGFNPAAFVQEFNIPERYIPTLLISVGKAAKPAYASGRFPLEQVIVKNSF
>lcl|NZ_CP026520.1_prot_WP_042230831.1_1972 [locus_tag=PC41400_RS10120] [protein=hypothetical protein] [protein_id=WP_042230831.1] [location=2209537..2209806] [gbkey=CDS]
MSVLQNELTHLIFLAEVVIASRKKEVMEDTLQCLLYIIKSLPEVEVPDSVAEQIAHLTERIEEKLRQENERIQEIQGNLG
QLAKPNSIA
>lcl|NZ_CP026520.1_prot_WP_042230830.1_1973 [locus_tag=PC41400_RS10125] [protein=helix-turn-helix transcriptional regulator] [protein_id=WP_042230830.1] [location=complement(2209918..2210298)] [gbkey=CDS]
MYFYDKLKDLRKLKGYTIRELADRSGVSAAYISQLENGNRNIPSPDVLMKLSEGLNIPYAELMKIAGYLDKPESGEDKPK
APVNLRRFLHDNELIFDGIELTEQDKEWVERMLSALFWKERNSRNE
>lcl|NZ_CP026520.1_prot_WP_042230829.1_1974 [locus_tag=PC41400_RS10130] [protein=helix-turn-helix transcriptional regulator] [protein_id=WP_042230829.1] [location=complement(2210499..2210879)] [gbkey=CDS]
MYFYDKLKDLRKLKGYTIRELADRSGVSAAYISQLENGNRNIPSPDVLMKLSEGLNIPYAELMKIAGYLDEPESGRDKPK
APVNLRRFLHDNDLIFDGMELTEQDKEWVERMLSALFWKERNSRNG
>lcl|NZ_CP026520.1_prot_WP_042230828.1_1975 [locus_tag=PC41400_RS10135] [protein=helix-turn-helix transcriptional regulator] [protein_id=WP_042230828.1] [location=2211064..2211270] [gbkey=CDS]
MRIIAKSEEFIRRRMTLGLTQRELARQTGLSHAYISLLERSMKSVGPGAAKRLSVVLGKQVEELFHIQ
>lcl|NZ_CP026520.1_prot_WP_042230827.1_1976 [locus_tag=PC41400_RS10140] [protein=GGDEF domain-containing protein] [protein_id=WP_042230827.1] [location=complement(2211360..2212487)] [gbkey=CDS]
MWNDPIETWGPETASSMMLVILTLMIFISGRLYYTRRKKGYLSMTVSLILMLAHNLLRLYWQASGQLQGMSGFADDLLQV
VTFVLINLGVYQLYNPTRRRHYLYFYGLIAVALGISAFHFYPPSGGSPEQAALLADLPLELYIFVLIFLCAYIITPLIGQ
QGKFQAMMTLYFARQTAHMVNLYLFQGEQKWLMLAEWFIPVIYATILFFLLFDRVVELMQAIYQSSITDGLTKVYNRKYF
DKRVSQYIRLNLNVSVLFSDIDNFKKLNDTKGHHMGDEMLKQVASIMRQEADESGIVGRYGGEELVILLTDPDVKPAELA
ERIRKRIEEETIVTVSMGFSTYKKGLTGDKLIKQADEAMYKAKTTGKNKVVKFSA
>lcl|NZ_CP026520.1_prot_WP_042230826.1_1977 [locus_tag=PC41400_RS10145] [protein=glycosyltransferase family 4 protein] [protein_id=WP_042230826.1] [location=2212767..2213891] [gbkey=CDS]
MNILQALFFPPEQPGGVSSMVPYITEKFNKRGWDMELFSLPRRVRGKGTEPVTFNTFDWTRYAGNPIVDKYIQTYKDYVW
WTKLRIRKSYDLIHAHHPIAALAIKEVFPDTAVIVTVHSSYEGELLLNGKIEGGGPEQLFLTSIYGEMEARLDQILTVSN
SFKGYISEFIQDPDRIGVIPNGYDEKRFRPISHENEVAQLITVCRLVPAKGLETLLQACAMLKERGHPFVLHIIGDGPSR
TELEELAKEMNLYEEIIFYGYMLHPEEFMPFFDIFVLPSKAEAFGSVFAEAALCLLALVGTNVGGVAEQIENGRNGLLVE
SGDAPALSQALEKVVVDPTYRYQLARAASEKAKKSYSLNRVIRQLRGVYERYGK
>lcl|NZ_CP026520.1_prot_WP_042230825.1_1978 [locus_tag=PC41400_RS10150] [protein=DNA repair exonuclease] [protein_id=WP_042230825.1] [location=2213932..2215380] [gbkey=CDS]
MKALKFLHAADLHLDSPFKGMNSVPERIRDVIRESTFGALSRLVRLAVREKVDFAVFSGDIYDAGDRSLRAQLRFRSALA
ELDRAGIPALVIHGNHDPLDGRGADLEWPAGAYVYPADKVETVRIVKEDRGLIAEVHGISYASAAVTENLALRFRGGDAS
VFQVALLHTNVDGNPGYDNYAPCSKSDLVGRGIDYWALGHIHTREIVMDEKPSVVYPGNIQGRSIRECGPRGCYIVAVDE
NGSPALDFHALDGVRWQQESLSIAGIGTEQELQESLQDLLARLREKAEGRPTVARIFLEGRGPLHLKLQAGSTLSDLLRE
LRESELLALAGRSAAGYGKGGTGAGAEYAEAPHPADDNKEEGTRGSYTTAPSGGVFRSAAAEVPFVWIESIEARTGQDID
FELLLREESFIGDLLRLAEELSADEGKLREFAEEALGSLLAQNGASRLFGGADGLLEDTGDWLEAARELAVDQLMPDGRR
AK
>lcl|NZ_CP026520.1_prot_WP_042230824.1_1979 [locus_tag=PC41400_RS10155] [protein=AAA family ATPase] [protein_id=WP_042230824.1] [location=2215377..2218748] [gbkey=CDS]
MIIREIRVRGFGSLQNRTYRFESPVTVLYGPNEAGKSTLLGFIRAVLFGFPTRAQAADRYEPDGGGSHGGALLLEDKQGR
TVLIERYADHGGPGRPPAAGSVKVTLEGGAEGGEELLRHLLGGLSAELFRSLFAFGLGELQELRTLQGDDIGGYLYGAGF
GVSGSAVVEAERKLAARADVLFRPRGRTQELNRSIRAYEELRSSRNRSADEISRYEEATAEWKRLGDRLNGLDAARAETA
AELEWTRRCGKAFEPWLKLRRVQAEMADIREIPGFPLHAAARYEAIRQERDEFENERRQYEEESAALLRAVEALPVREEL
LEDAAELAALEEEAGTYRSGQLAAASSETEKAQLNEALAALLRQLDEQWSREDLAEFPVSVGMKEQIREARDSFAKWDGE
RALLDSELSRVFGRLEEQQSELAERQLEWSEWLTLSGLREDAPGARTAAESGLLRRLSSDYAQWRLLKSEEEQRDRIREE
ERRRQEEFEAWSDRRRSDSDRTRKKTAYAAGAIAVLLPAALLLSGQGPAAAVVFVVLAAGTAWLMFGSPDRARGNSRSPS
TLLPEVRTEGRGRARRGRIASSGDWLSAPAQPGEADRTDGEDVRLEADKAALRQRLQQNLALLLSSRGEGSREAAAARTA
GANGFFDEASAPRRMPLPPWPETLSWLDRELESWLTGTDLWAQHRSEMERRRAKLEELKQSIQSLEKQQAAVSSRLRSLE
ERAASGQERWSGWLCERKLRPHLSPEAALESLQLIARGHDLLQRLAGCESKLAALQATSAAFEARTVRLLGEAEAADPLL
GLRRCAERLAEQQGLLARRERLAAEAGEARRREAAAAQKARHAAERLQALLSEAQAAGEEELLRQAALQERLAQLAAEAR
LLSGALNAHAGVARRERLDEALSGADEGSLASREEELAGSLSGLEAEANRLRDRRGRLAGELEKLEGGSERGELLQRTEE
QLAEVRDLGGKYAVLAMAALLVKRTRERCERERQPGVLRRASACFAAMTGGRFTQVVAPFGQQRLYAVRSDGRQVDSSRL
SRGTAEQLYLSMRFALVEEFTGGASLPLVFDDIFVNFDKERMEGALRIIGQLGGKRQVLLFTCHEHVRDAVTGISPSAGV
LHV
>lcl|NZ_CP026520.1_prot_WP_042230823.1_1980 [locus_tag=PC41400_RS30235] [protein=Ger(x)C family spore germination C-terminal domain-containing protein] [protein_id=WP_042230823.1] [location=complement(2218842..2219183)] [gbkey=CDS]
MTDQNDSRKPDGRPTLPAPPTLNADQWDIPLAADTDGVIQLFQELLPTDEFQWILTDNKLLGGTKEQGLDILGIGEQFRM
RGWDTTNWEELIKKLEVTVRVHLKILSPEGKRG
>lcl|NZ_CP026520.1_prot_WP_042230822.1_1981 [gene=pfkB] [locus_tag=PC41400_RS10170] [protein=1-phosphofructokinase] [protein_id=WP_042230822.1] [location=complement(2219553..2220482)] [gbkey=CDS]
MITTVTLNAAIDKTYSVKSFEMDRVSRVTDMIAVPGGKGINVARVARLLGEPVTAAGFVAGFNGQFIRSGLDTEGISHDF
TEVPGESRLCLTVLDADNGTQTELLEPGPVIAENSLLTLKEKVKELAGRSTHVCFSGSLPLGCPPSLYAELITLAREAGA
EAVLDASGDALAAGLEAMPLLIKPNEHEIVRLLGRPASSDDDLAACIRELMQRGIRHVVVSLGARGALAGSSGRVRRVAI
PAIRAVNPVGSGDAMVAGMVTALRQGRSDADVVRFGAACGSANALQKQAGVVTDEDVERIFRDITVTDY
>lcl|NZ_CP026520.1_prot_WP_042230821.1_1982 [gene=fba] [locus_tag=PC41400_RS10175] [protein=class II fructose-1,6-bisphosphate aldolase] [protein_id=WP_042230821.1] [location=complement(2220508..2221377)] [gbkey=CDS]
MAHLISSTHLLQTARQNGYGVTAFNVHTLEMLQAVVEAAEELQAPLILQTTVGTVKHLGPDYIVAAAAAAAKRSSVPIAL
HLDHCTDYELIVQCIREGYTSVMIDASMHPFEENVRQTQEVVKVARAAGVNVEAELGKVGGVEDDIVVADEDALLADPDE
CVRFVELTGVPTLAPAIGTAHGIYKGEPKIAFGKLEEIYRKVSLPLVLHGGSGIPEEQVKRCVSLGMAKMNVATELRIAF
SDAIKKVFQANPDENDPRKYMVPAKAAVKELAQAKMRLTGCVGRAKDFI
>lcl|NZ_CP026520.1_prot_WP_042230820.1_1983 [locus_tag=PC41400_RS10180] [protein=SIS domain-containing protein] [protein_id=WP_042230820.1] [location=2221469..2222527] [gbkey=CDS]
MSNIQGQLTYTEISAQSAALQAAWDQLQKQSDWVKRYIGNDQYEEVVFIGSGSSYYQSITMAATFRKWTGRSASAQPSSD
IFLLRDATLPRGKKLLIVGVSRSGESHEVLLALDSVKNEDHIDTCGITCHETSKMFGMTECLLSPLGKEKSTVMSKSLSS
MTFMMQAAIAMAAGGEAYLQELKTVLELDEELVKASDSSIKEWVSAHNFNKTVYLGLGALGGLALEACLKLKEMTYTWTE
AYGTLEFRHGPKSIVDSGTLVCFLLSQSAREYELKVARELKEYGATILLLTAERGEDTDFADLVLELGGAALSDEARSVL
YLPAVQYNGYYNAVKRGLNPDDPRNLTQFVKI
>lcl|NZ_CP026520.1_prot_WP_009674761.1_1984 [locus_tag=PC41400_RS10185] [protein=DeoR/GlpR family DNA-binding transcription regulator] [protein_id=WP_009674761.1] [location=2222552..2223340] [gbkey=CDS]
MNDTTPSKGDLRRQQIMTILKQQGRITIGEIIEHFQCSEATARRDLDLLEKKGEVVRTIGGALFEGLTAVRETSFAEKKQ
QLWLEKEAIARRAAGLIEEGDSVCLTGGTTTFLIARELKQRQGITVVTNAVNIAMELADSEGLQVVVVGGVMRSKSFELS
GPLAEKTIEHLNIEKLFLGVDGVSADKGITTFSELEAQTARLLISRTQHTIAVFDHTKVGKASLFSIAPLADVNVCVTDA
RLEPDMEKVLEQHGITTLYANA
>lcl|NZ_CP026520.1_prot_WP_042230818.1_1985 [locus_tag=PC41400_RS10190] [protein=ROK family protein] [protein_id=WP_042230818.1] [location=2223359..2224333] [gbkey=CDS]
MSFFVGVDLGGTNIVCGLLDEEFNVLAKSKQPTEAAKGSDYVLEKIAVMVETLLREQNIAPEQLKAVGIGTPGFIDPVRG
VCVFASNLRWNEVPVSDQLGARLGVPVFIDNDVRMYVYGEAMKGPGRGYEHVLGITLGTGMSAGLVNNGQLYYGGGFMAG
EIGHIRVDDAESTPCGCGLVGCLETIASASGIVRQVREALAAGRESILGQWFPGEEAASVTAADVSRAYDEGDRLAIEVM
NHTGKLLGRGLSYAVTMYSPDALIIGGGASLAGERLFKPMREELQKSVYKGYWERLTIHPGTLIDDGGVIGSAAFAANRA
AGIV
>lcl|NZ_CP026520.1_prot_WP_042230816.1_1986 [locus_tag=PC41400_RS10195] [protein=hypothetical protein] [protein_id=WP_042230816.1] [location=2224538..2225056] [gbkey=CDS]
MSEKKPAKPGDREITYQVGRRKGRIRTAKAKAPVQDEKLPGPKREPAKPKKSASERKKAKTEAERAYSRSRQVEGMEAAK
SSEYIEHPSNPSSAAASGESTNRAPNPSNPAVRTYPEEKATPVPGENSRLMELLWRSTPLPGWRAANMITWVLPVAAAAC
TLWVVSKLLTFG
>lcl|NZ_CP026520.1_prot_WP_129112468.1_1987 [locus_tag=PC41400_RS10200] [protein=hypothetical protein] [protein_id=WP_129112468.1] [location=2225177..2225497] [gbkey=CDS]
MWKRDVRGAAPEQHAAKFLQRITPLDYAQFALMAIGIIMLLADLVAVLRDAKAFPDYHIGYLLCGFLFVFLGMLMSFNRL
LLTVRAWRSSDGAVSPHHHGQPHRAD
>lcl|NZ_CP026520.1_prot_WP_042230814.1_1988 [locus_tag=PC41400_RS10205] [protein=membrane protein] [protein_id=WP_042230814.1] [location=complement(2225444..2226484)] [gbkey=CDS]
MASKTWTQTAQVAFTYVGTVVGAGFASGQEILQFFTRYGAAASLTIALASVLFVWLGIKMMLMAHDVRAESYEDLNIALF
GPKIGRWISLFTLFVLFGISSVMLAGGGTVFSEHLSLPFQSGLLVTAVLSYAVITRGIGAIMAVNTVVVPIMATFSLILV
FYTNKLPTSQAWLTLGTDHSPLSAWTAPLLYVGFNLATAQAVLVPLGAQIKDRRVLFRGALLGGLGIGLLLFAAHYALSA
QMPGIRQFGIPMARIMDGLGTVIPFVYLLVIYGEIFTTYLSNTYGLTLQLQQRTGLGRRTVLLLLIALSYIVSLIGFKPL
LSALYPLFGLVSSVWLAVMMWRNRSV
>lcl|NZ_CP026520.1_prot_WP_042230812.1_1989 [locus_tag=PC41400_RS10210] [protein=DedA family protein] [protein_id=WP_042230812.1] [location=2226594..2227205] [gbkey=CDS]
MKETIFELISNYGYTALYALLAAGIVGLPIPDETLMTFVGSLTMDTGPLNFTHSLIVSYLGTMTGMLISYTVGLRLGKPF
LYKIGKWVRVKQERITRTELWFQKYGMWAVFFGYFVPGLRHFTCYLAGVSGISLWRYLLFAGSGALLWCCTFLSLGHFIG
ANFEAVLKGFHTYLGICLMGLAVAAAVGFLIYYLVRRKRKKNS
>lcl|NZ_CP026520.1_prot_WP_042230811.1_1990 [locus_tag=PC41400_RS10215] [protein=hypothetical protein] [protein_id=WP_042230811.1] [location=complement(2227257..2227913)] [gbkey=CDS]
MKNKLRRAAMIGLLISVLPAAAGCEAVHTFWGKETAGPSANQAESNGNAQPSPSGAAGQTPAPDRSAAVSSNENTEGPVK
AKPVVTPENTYRVNNPTLMGVTLGTPKDEVLNLFGKAKNMFVMDEDPQSVTVYEYGSFSVGFNVFDRLEFVQIQSADIDP
GLGGLRLGQTADEATDVLGKPNNRTDYVLAYKAQNTVLKLDLDPDNHTIQSIKLFPSS
>lcl|NZ_CP026520.1_prot_WP_042230810.1_1991 [locus_tag=PC41400_RS10220] [protein=xanthine phosphoribosyltransferase] [protein_id=WP_042230810.1] [location=2228257..2228829] [gbkey=CDS]
MELLKQRILEAGSVVSDQVVKLDAILNHQVDPALIKEMGREFARRFVESSPTKVLTVESSGIPLSFAVASELGVPFIFAR
RKKSVTMGEDTYCERVPSFTKGIVTDIMVSSQYITANDRILIIDDIIANGDAARGLIRIVERSGAELVGVGIAVEKAFQA
GGRAIREQGIRTESLVTITSLDQGNIVFAE
>lcl|NZ_CP026520.1_prot_WP_009674785.1_1992 [locus_tag=PC41400_RS10225] [protein=hypothetical protein] [protein_id=WP_009674785.1] [location=2228929..2229255] [gbkey=CDS]
MGNEQIPAEYLLEKLSEAKNHFERALDCKHTEFDDLYPYMIEHPQFFWYKRYVAWSELLTVVKFFTELSVPWESQFTVQQ
VDYIKKRVMSSKVLDCWFENTDSKEHVG
>lcl|NZ_CP026520.1_prot_WP_042230809.1_1993 [locus_tag=PC41400_RS10230] [protein=hypothetical protein] [protein_id=WP_042230809.1] [location=complement(2229304..2229615)] [gbkey=CDS]
MDGKKTYYVSVGAKEIVDEADEINFDFEIKATGEEIDKLAELFEVMDESDEATHFRALLPFKEYHKDKENDAYDYDLKEI
YRTLYNLGTEKTRKHIEEMNVLN
>lcl|NZ_CP026520.1_prot_WP_042230806.1_1994 [locus_tag=PC41400_RS10235] [protein=hypothetical protein] [protein_id=WP_042230806.1] [location=2229906..2230154] [gbkey=CDS]
MISDEQLDRYRIDGTLLRIVRDADPRNDIRGYVVAWNESSVAIRKRSRKLVQLDRKYVYQPYTESRPAEYTLPRQTDEGE
EA
>lcl|NZ_CP026520.1_prot_WP_042230803.1_1995 [locus_tag=PC41400_RS10240] [protein=adenosylcobinamide amidohydrolase] [protein_id=WP_042230803.1] [location=2230159..2231151] [gbkey=CDS]
MNTQEPLIQTVSRPAAGISAAVNEGAAGSYLLLEAPSPLRTLSSAPWGGGFGYHRTIINRKVDKTYACGDPVTEMNGFLE
AAGLNSEETACLLTSVHMSGLGQSHLRMDDPASAKVRADDEAARSLFDTSDSKLDVSSQADREAQSFLETFDERPDTRGR
GKACLSAAADGDSANPVNGIDAAEPAAAAAYATAAAAAADTGIAVSAWVTVGYSNMARAGRGLPASALFPGTINIIVAVE
GELTDEAMVNAVITATEAKAAALQDLGVALEDGSGATGTTTDAMLIASTQRGRLHRYAGTATYLGHMIGRSVYEAAMQSG
MAYKKALALS
>lcl|NZ_CP026520.1_prot_WP_042230801.1_1996 [locus_tag=PC41400_RS10250] [protein=membrane protein insertase YidC] [protein_id=WP_042230801.1] [location=2231824..2232510] [gbkey=CDS]
MSLLQPFISMLDRILNPLAAFTNDWGLAILLFALLVRSLLFRVNLLSARQQVRQIRMQPAIRQLREQYANQSRKLGEEML
RLYSRYGIKPGLAFAAGLIQMPIFMSLYPYFSLHGAQMTTMLIPWMQSLGVPDPAHALPVLYGIIGWLGMLISLVPDPAA
SQPALLKVGMPLAIMAVSLTVIWRSPAALVLYWAGSGIVTILERLFFRSSYGQKLLHKGIPVPELIKD
>lcl|NZ_CP026520.1_prot_WP_042230799.1_1997 [locus_tag=PC41400_RS10260] [protein=DUF1772 domain-containing protein] [protein_id=WP_042230799.1] [location=complement(2232832..2233290)] [gbkey=CDS]
MSGFIVAFAIAIMPGLNETGALSAVHSMNAINHAIGGSPLFFILFGGTGMLHIAGLVIALKNLKLPFVWLIICAAGIYLC
GVLFVTLCLNIPLNKEMAGLDLTISRNDLVAGDILARWVFWNQTRAVSSLMSVLLLAIYLRSIYFMNHGFQS
>lcl|NZ_CP026520.1_prot_WP_042230797.1_1998 [locus_tag=PC41400_RS10265] [protein=type II CAAX endopeptidase family protein] [protein_id=WP_042230797.1] [location=complement(2233371..2234177)] [gbkey=CDS]
MVINNKSSNHSPWIFFLLVFILAIPFWLFGLLTQQLLQKEIPINLPISALMFVCPMIAALILVYRNNGANGIKQLLSKAF
KIKIKRKIWFAPIFFMEPFIMFLAYGSMRLMVGSIPELDFSLLKILALFFVFLIAGICEEIGWQGYAYDTLENRWNAVGA
SIMLGVVWQIWHIIPHLQLHPPAWILWQCAGSVLLRVLIVWVYKNTGKNVLAAAIFHTMNNVCSFLLPNYDSNIAPISMF
IVTAFVVLIVTFLWGSKTLAQYRFIHNR
>lcl|NZ_CP026520.1_prot_WP_042231177.1_1999 [locus_tag=PC41400_RS10270] [protein=TetR/AcrR family transcriptional regulator] [protein_id=WP_042231177.1] [location=complement(2234237..2234815)] [gbkey=CDS]
MSKERIIKAAIEVFSENEYHRASMDEIALRAQVAKGTLYYNFPGKSQLFKTVVKRTFEDFIRRIKADLQSSIPLKDKIQQ
TIRHHLDLFLESRHLSHIVFNDISNGFEEDVLDELKLLREEHLNFLATILEEGKCDENYLRDLDSHIAAVGMIGTLESTS
KYFLSHQNKYSRDDLERQVFTMITMSLFNSIE
>lcl|NZ_CP026520.1_prot_WP_053228730.1_2000 [locus_tag=PC41400_RS10275] [protein=GNAT family N-acetyltransferase] [protein_id=WP_053228730.1] [location=2235171..2235878] [gbkey=CDS]
MNNPFLQTDYELMETQVNVLFKHNAVGRMTAINEPSIKSAPRFFMGITKQGNVVRYHETLDEDVKNELEQFAEKAPWSDA
PNRVNMADIIRILSKERPVDHVYLGPAYVFPDMRGRSTQAIKITQMNIELLEPYFSDFVNDVIYGQPAFAVVQDQAAVSL
CSNARSSSIAAEASLHTARDFRGRAYGLEAATAWAAEIQNQGRIALYSTAWNNFASQAVARKLKLRQYGMDLHFS
>lcl|NZ_CP026520.1_prot_WP_053228729.1_2001 [locus_tag=PC41400_RS10280] [protein=Bax inhibitor-1/YccA family protein] [protein_id=WP_053228729.1] [location=2236259..2236912] [gbkey=CDS]
MERTMTYSHQGSLSHILRTFALSLLISFVGMLVGAMIVPPSIIPLFIVVELVMIVAAIIFRMRGKNIGYGFLYAFTAISG
VTMYPVIMAYGSMIGANLVSGAFLATAVIFGGLAWYAARSEKDFSFLGGFLFAATIGLILMGVLSLFVNFGSTLNLLLSV
GGILIFSGWVLYDVAQYREGVAAEEVPLAAMNLYLDFINLFLYILRFIASIVGISRD
>lcl|NZ_CP026520.1_prot_WP_042230795.1_2002 [locus_tag=PC41400_RS10285] [protein=YjdF family protein] [protein_id=WP_042230795.1] [location=complement(2236990..2237400)] [gbkey=CDS]
MKLTVFYDGQYWIGIAEEERNGRLLAARHIFGTEPLDGEVLDFVRHDLLRLLENVGKGVRAAPMTERKLNPKRLARLASR
ELAQKGPSTMAQAAIQADLEYRKKVRKTLSREEREADAARKREIARQKAKDKHRGR
>lcl|NZ_CP026520.1_prot_WP_042230793.1_2003 [locus_tag=PC41400_RS10290] [protein=YqzE family protein] [protein_id=WP_042230793.1] [location=2237682..2237882] [gbkey=CDS]
MAKGGEELVKYITEQVVHYIETPRQVRKEARVRHKETRESWSVHWFGMIPLSVSMIIKAVRRRSKD
>lcl|NZ_CP026520.1_prot_WP_042230792.1_2004 [locus_tag=PC41400_RS10295] [protein=hypothetical protein] [protein_id=WP_042230792.1] [location=complement(2238039..2239115)] [gbkey=CDS]
MNRTALWTAALLALVVWLAAGPLSFSQAESLLAQDAQDSLAGPATRMKPETGRASLDKRYTILPGTRPAEVLADFTANEP
SDTPGAPNAGTVLNEHIVRWIDSLAGQKGFEGWKRAKWTVAPLGPGTHGWIVLLTDNGRETGYLIVSAAENGTLQLIEYG
TGEHPLFSTRTLYRTLVQHAIIPASMSYQAWLDSAETRLRRLYTDAFHAVWTVPAGNNRILVDAKTGEIRTFQNYSPSSD
SSLLGLHGAAFEHTPGTPIEVTPSDPYEEIAWLTRKALPIRSYEELTPYLAAGKSLTFVCECFGGRLTLPLPVTGALRWI
GGAAFIAFEQEGRRFVLLDEAVSLGQFYPTDGISSSSP
>lcl|NZ_CP026520.1_prot_WP_042230790.1_2005 [locus_tag=PC41400_RS30575] [protein=YqhG family protein] [protein_id=WP_042230790.1] [location=complement(2239231..2240514)] [gbkey=CDS]
MNTAQVERFVMRFLEAYNCRIIEKTKHAVIVNLSPEADKELTGRSYYWSFVERTGVEPETMTYRFVFDPEGDAAASAAAR
PVPAGPRNGTLGAAGPQPGAAERQAQEAAGTGNGPQGGLAAGTAGSAGPVAVAGAVSAAPGAAGGVGQAAGGGTVPAGSS
GTPSPQGTLAAGAAGSAASAGSVPGGAAPAPGSGGSPPEAIRSADSILGRYFGTAPAPPVGRVPRDEVTFGSRRLQQLFA
TALNNGRFTRLFEAPPQAPGSPLPGRSAPLTYCSWMNVNYKVEYACDMKRSEVHSLAIQMTTGEIREEFHPSMLTRKLAP
KLPANIHLLGDNLTLSRAASILEDYLEQKIRRTDLRWADRANQVLAEEHQLVRLYYEGMLTAAEPEAREELEIRYAARQA
EVDWQYRPRVHVSVINCGLFHLRADER
>lcl|NZ_CP026520.1_prot_WP_042230788.1_2006 [locus_tag=PC41400_RS10305] [protein=SNF2-related protein] [protein_id=WP_042230788.1] [location=complement(2240495..2242294)] [gbkey=CDS]
MNNQTTEHIRPIHEGLSMHTDREWFLDWESRVVRNGPWDNWTLYQLAYEAEKSSLIHSFDELLCLKHVNGIEPMSHQIDT
AKKVLYDMRGRAILADEVGLGKTIEAGLILKEYMIRGLVRKALILVPASLVLQWVRELNQKFGIAASAQKKSYMWQTCDV
IVASMDTAKRDPHREIVLGLDYDMLIVDEAHKLKNKKTTNYQFVGELRKKYCLLLTATPVQNDLKELYNLINLLKPGQLG
AQTNFQANFVMAKRMPKNEGVLQQELSKIMIRNRRGDGNVEFKKRIVKNITLTLSPEEQALYDGVTDFIRGQYEQNQGQP
SNTLALVTLQREVCSSRDAVFITLVNMFKKTEENSPVRSKIWELVERIRSIQANTKAETVLNLIREINDKVIIFTEYRAS
QEYLLNYLKENKITAVPYRGGMNRGKKDWMMDLFRTRAQVLVATEAGGEGINLQFCHHMINFDLPWNPMRVEQRIGRVHR
LGQTDDVRIYNLSTRNTIEEHILNLLHEKINMFELVIGELDSILERLEKKSASLESSLFKMMMESRSDEEMRRQIDRLGD
SLTGIREELEPENERKAKIGSILSAIGQNVGVNHEYRPG
>lcl|NZ_CP026520.1_prot_WP_042230786.1_2007 [locus_tag=PC41400_RS10310] [protein=biotin/lipoate A/B protein ligase family protein] [protein_id=WP_042230786.1] [location=2242633..2243475] [gbkey=CDS]
MTTWRYMHTGNRTAAENMAIDEAILIAHSEGRVPPTVRFYGWGPAALSIGYFQKAQKEIDLDKVAGQGLGFVRRPTGGRA
VLHDAELTYSIIVSERYPGIPTSVTEAYRVLSEGLLLGFRKLGLAAEMVNLATEEEKRKYESLGSAACFDSPSWYELVVE
GRKVAGSAQVRQKNVVLQHGSILLELDVEQLFGLLLFSNERIAERMKQSFVQKAVAINDLLRSAGLAPVTLGDVETAFRA
GVAEGMGIELENGALTPYELELAEELAREKYSSDEWNLKR
>lcl|NZ_CP026520.1_prot_WP_042230784.1_2008 [locus_tag=PC41400_RS10315] [protein=AAC(3) family N-acetyltransferase] [protein_id=WP_042230784.1] [location=2243505..2244323] [gbkey=CDS]
MAIDTIIERTGKRPHTAESLAEDLRRLGVKEGMTLLVHSSLSSLGWVCGGTQAVIAAIRDALGSQGTLMMPTHSGDLSEP
SAWNNPPVPEEWWDVIRSTMPAYKAEETRTFFMGALPEAFRTYPGVMRSSHPQVSFAAIGPQAQKLTGSHSLAYGLGEGS
PLARLYELDGHVLLLGVGHGSNTSIHLAEYRAEYASKKEVTNGAPVLDGEGVRQWVEFPDVGIDSDDFERAGADFEVRNS
PVIRGKVGSAEAALMPQRPLVDYAAVWMSLHR
>lcl|NZ_CP026520.1_prot_WP_042230782.1_2009 [gene=dat] [locus_tag=PC41400_RS10320] [protein=D-amino-acid transaminase] [protein_id=WP_042230782.1] [location=complement(2244636..2245481)] [gbkey=CDS]
MMLFGNQLVPDEEVRISPLDRGYYFGDGVYEVFRIYDGRLFEVQGHMERFSRSMAEIRISLPYPLEELENLLNRLTETAG
IPDGLTYVQITRGAAPRSHPFPAAAAAEPVVTGWCSPFLRPLKDLENGISAIVREDIRWHRCDIKSLNLLPNVLLKQEAL
DGGAGEVIFSRGGVITEASASNVMIVKNGIVKTHPADHHILRGVTRSLVLRLAEGLGIAVREEAFRAEELASADEVFITS
TGSEVMPVVLVDSRPVGDGKPGPVTRLLQQAFEASIPAGSR
>lcl|NZ_CP026520.1_prot_WP_042230777.1_2010 [locus_tag=PC41400_RS10325] [protein=adenosylcobalamin-dependent ribonucleoside-diphosphate reductase] [protein_id=WP_042230777.1] [location=2246030..2248648] [gbkey=CDS]
MKTMQRKSLEGLSEKIFLDRYALKDADTNNTKVGDVVLVLTKDDPKFPAKEVGEITARDGRNVTVKLRSGETIESNIDRL
TLNIEKTPDELWDRLAKAMASVEATPEKQKEWTEKFRYILDDWKLVPGGRIAAGADASDELTLFNCYVIPSPHDSRGGIM
ATLSEMTEIMSRGGGVGINLSSLRPRRAIVKGVNGSSSGSVSWGGLFSYTTGLIEQGGSRRGALMLMINDWHPDLLDFIT
VKSTMGQITNANLSVCVSNGFMKAVKEDLDWDLVFPDTTDPEYDEVWNGDLNEWKKLGKAVKVYRTVKAREVWHTIIESA
WRSAEPGVVFMEYYNQMSNSWYFNPIICTNPCGEQGLPAWGVCNLSAVNLSKFYDEENDDVAWDELGKTVRYSTRFLDNV
IDKTPYHFEENRENQQGERRVGLGTMGLAELMIKLRIRYGSPESLEFLDKIYGFMAREAYLASSEIAGEKGSFKHFDTEK
FLQSGFMKNMHEVYPEVTESVRKNGMRNVTVITQAPTGSTGTMVGTSTGIEPYFAFEYFRQSRLGYDKQFVPIAQEWKDA
NKDQDLPDYFVTSMTLSAEDHIRVQAAIQRWVDSSISKTANCPADFTVEETERLYELAFDLGCKGVTIYRDGSRDVQVLS
TGTDKDKKDEKKAEAAVPAPEVPTEAPSVAVAAEPVVQADAISGNVFDKEYKRRPQILRGATYKINTPFGMAYITINDMN
GSPSEIFLNVGKAGSDVFAMAEALGRVCSLFLRYGDHGNKVNLLIKHLKGIGGSGAIGFGANRVESIADAVAKALEMHEE
AMAGEAGAEAYVTATKEVVHEAPDALGHAAPSAGAAPAGLSSLDLCPSCGSASLVNSEGCKNCTNCGYSRCS
>lcl|NZ_CP026520.1_prot_WP_042230775.1_2011 [locus_tag=PC41400_RS10330] [protein=hypothetical protein] [protein_id=WP_042230775.1] [location=2250095..2250388] [gbkey=CDS]
MFNLEEIKKRENYFRSLLNADTSGKNFLDKNFEEVKQYLSDKKDLIEAYVRLQTEIEYVKPVHLKERKGNITKIGYRGAE
YAMIHPTAMRTKGGIFK
>lcl|NZ_CP026520.1_prot_WP_241688836.1_2012 [locus_tag=PC41400_RS10335] [protein=epoxyqueuosine reductase QueH] [protein_id=WP_241688836.1] [location=2250430..2250963] [gbkey=CDS]
MIKDWEAKNERPKVLLHSCCAPCSTSSLEFLAEHADVTIFFSNSNIHPKSEYMRRSNEQKVFVEKFNNRTGHTVGFIDDD
YRSNVFFKMVKEEGLAEEAEGGLRCSACFDMRLDRSAEVAQDGYQRSIEMCNEYDVFRQCYCGCSYAADKQGVDLRAVNK
DAINYLKENNLSLKKEI
>lcl|NZ_CP026520.1_prot_WP_042230773.1_2013 [locus_tag=PC41400_RS10340] [protein=serine hydroxymethyltransferase] [protein_id=WP_042230773.1] [location=2251028..2252275] [gbkey=CDS]
MEQLRKQDPKIVEAMNLEVGRQRNKIELIASENFVSEAVLQAMGTVLTNKYAEGYPGKRYYGGCEYVDIVEDIARERAKE
LFGAEHVNVQPHSGAQANMAVYLAALKAGDTVLGMSLAHGGHLTHGSPVNASGILYNFVSYGVRESDARLDYDEVRKLAF
KHKPRMIVAGASAYPRTIDFESLASIANDVGALFFVDMAHIAGLVAADQHPSPVPHAHFVTTTTHKTLRGPRGGMILCRK
PWAAAIDKAVFPGTQGGPLMHVIAAKAVSFGEALQPDFKVYAEQVVKNAKALSEALLAEGVNLVSGGTDNHLMLIDLRSL
DITGKDAEHLLDEVGVTVNKNAIPFDPAKPMITSGIRIGTAAVTSRGMDEGAMQVIAKIIALTLKNPSDEAIHEKVRDMV
KDVSAQFPLYPGLQY
>lcl|NZ_CP026520.1_prot_WP_053228728.1_2014 [gene=pabB] [locus_tag=PC41400_RS10345] [protein=aminodeoxychorismate synthase component I] [protein_id=WP_053228728.1] [location=2252643..2253629] [gbkey=CDS]
MDSTMWQVMFRTSVPPAFDKHFSVCEPDEEFPLVWFGIYEKPIHDDDIGQKGGYSVSDWRTDTTRNEYRNNIEHIHAAIA
RGETYQVNYSTRLKAEFSGDELAFYEDLRDPLTNYSAYLNVGRYRIVSISPELFFHWDGDTIRTKPMKGTLSKPGHQQTL
FESEKNRTENVMIVDLLRNDLGRIANTGSIRVPNLFEIEEYPTLYQMTSTIEASTKPGVTLFDVFQAMFPCGSITGAPKI
STMKIISSLEKEPRHIYCGSIGLAVPGEGITFNVAIRTVLLDTHTQQAVYGVGGGVIWDSTTEDEYDEMRTKSKVLTYRY
DELEINAE
>lcl|NZ_CP026520.1_prot_WP_129112469.1_2015 [locus_tag=PC41400_RS10350] [protein=NUDIX domain-containing protein] [protein_id=WP_129112469.1] [location=2253663..2254151] [gbkey=CDS]
MNYEFPGALPNQNIILTRTSLIEEANCVLIFAFYKGKLVLVRHHCRGWELPGGNREEGESIIQTVIREMYEEAGGELESI
ERIGQYLIYEQGHLVFVKNIYISKVSVLRELPNGFETDEVMLLETIPKQIEIMEDPAFSPYMKDSVYLIVAEWIKGHPFT
SL
>lcl|NZ_CP026520.1_prot_WP_042230772.1_2016 [locus_tag=PC41400_RS10355] [protein=LysR family transcriptional regulator] [protein_id=WP_042230772.1] [location=2254236..2255177] [gbkey=CDS]
MTLQQLKYILTIVSCGSISEAAKRLFISQPSLSNAVKEIEMEAGIEIFLRSSKGIVLSSDGAEFLSYARQVVEQAELLEQ
RYMNRKPSKKLFSISTQHYAFSVQAFVGLLKELNVDEYECTLRETRTHEIIEDVRNMKSELGIIYMSEFNRKVIQKILKD
NELRFHPLFQAEAHVFISEEHPLASKSLLDIEDLDDFPCLAFEQGEYNSFYFAEEILSTRTYKKKILVSDRATLFNLLIG
LNGYTISSGVLNKDLNGEQIKSVRLRTDESIQVGYIVNQKASLSQLAAAYVLKLKTVITDFGYPIINQEDELA
>lcl|NZ_CP026520.1_prot_WP_042230770.1_2017 [gene=metE] [locus_tag=PC41400_RS10360] [protein=5-methyltetrahydropteroyltriglutamate--homocysteine S-methyltransferase] [protein_id=WP_042230770.1] [location=complement(2255198..2257468)] [gbkey=CDS]
MKTSIIGYPRVGALRELKFASEKYFKGVLPVEELQQTAAQLRTANWKLQQENGVDFIPSNDFSFYDGLLDTACLLGIVPQ
RYQALGLSQLETYFAMARGYQGNEGDVKALAMKKWFNTNYHYMVPEIDDTTSIRLAGSKLFDEFTEAKQLGIATKPVLIG
AFTLLKLARFTGSKKAKDFVPAVVEAYTAVLNQLNTLGAAWLQLDEPALVTDLTAEDIALFTELYNGILSAKGSVKVVAQ
TYFGDVRDCYQVLQALPFDGIGIDFVEGKQSLDLVRKHGFASDKVLFAGVINGKNIWKNQYKQTIALAHEIGKHASQVVL
GTSCSLLHVPYTIKHETKLTAANKQYFAFAEEKLRELAELKSILEQASPEASAFYTENEKLFSASRLTNDNSVQKRVAEL
TDADFTRLPAFAERETIQKAKFNLPPLPTTTIGSFPQTADVRANRSAFKKGNITEQQYKQFNFDRIAECIAQQEEIGIDV
IVHGEYERNDMVEYFGECLDGFIFTENAWVQSYGTRCVKPPVVWGDISRSKPITVEYSAFAKSRTNKPVKGMLTGPVTIL
NWSFPREDISLKESALQIGLAIRDEVLDLEANGIEIIQIDEAALREKLPLRHSDWQSEYLSWAIPAFRLVHSGVKAETQI
HTHMCYSQFNDIIRDIDAMDADVITFEASRSDLAIIDAINECGFRTEVGPGVYDIHSPRIPSVEELKQGLDRMLDKIEVS
KLWVNPDCGLKTRGVTETWASLKNLVQAAKDVRATL
>lcl|NZ_CP026520.1_prot_WP_042230768.1_2018 [locus_tag=PC41400_RS10365] [protein=glutaredoxin] [protein_id=WP_042230768.1] [location=2257838..2258062] [gbkey=CDS]
MKLYTKTVCPKCMWIKSEIARSGIAVEIVNIDHDEAARERLLEAGVMSLPVMEAGGEFIFDTNEMVKLMEQVNV
>lcl|NZ_CP026520.1_prot_WP_042230767.1_2019 [gene=nrdI] [locus_tag=PC41400_RS10370] [protein=class Ib ribonucleoside-diphosphate reductase assembly flavoprotein NrdI] [protein_id=WP_042230767.1] [location=2258059..2258430] [gbkey=CDS]
MIAFASRTGNVRYIVSRLRLPAIEIEEDLMLAESFLLFTYTDGLGEIPAKVKLFLERNGKLCRGVIVSGNSNFGHNVFGG
AGDAIARQLKVPLVRKIDLRGFPEDYEVIHDYYEQCFRKEHIG
>lcl|NZ_CP026520.1_prot_WP_084706616.1_2020 [gene=nrdE] [locus_tag=PC41400_RS10375] [protein=class 1b ribonucleoside-diphosphate reductase subunit alpha] [protein_id=WP_084706616.1] [location=2258427..2260520] [gbkey=CDS]
MKSYLKLNNEVLNQYSRTGKLDLAKDKEATRRYFLEHVNAKLRYFIDTREKIRYLVDEGYYESEFLEQYDMSFIERLYEK
AYAYEFRFPSFMSASKFYESYAMKSRNGEEILEKYEDRIVITALYLAQGDEHLAERAVEVMMSAYQPATPTALNSGKKAR
GELVSCFKLTMDDSMNSISENIGYCLELSRLGGGVGVNVTDLRPLGDPIKGILNRASGVMPVAKLLENSFSYSNQLGQRN
GSGVIYLNIFHADIENFISAKKPNADDKIRLATLSTGIIVPSIFFELMKRDKDVVLFSPYDIYKEYGKRMSEISISEMYY
ELLDNPNVRKIKRINARKLYTEVKKAQFESGYPFEVFDDQVNEKHPLRNLGRVKMSNLCTEILQVQSTSIINDHDKENEY
GLDVSCNLGSLDIHKAVQEDDFEQLIDTSMRLLTDVSMMTEIKNVPSVAKANRLMHSIGLGLMNLHGHLVSQGILYGSKE
SVQFIDAFMEAVNYYSLLASMRISQEKKATFHGFEQSDYANGSYFEQYVSKEVTELPEQVRKALGNVPVITQEMWRSLKE
QVMEHGLFNAYRLAIAPTGSISYIRSSTASIAPCTEKVEIRDYADSRTIYPMPFLNNDNISLYKEAYEIDQYRMIDLYAA
AQRHIDQGISMTLYVTDQWTTEQLARLYIYAWMKGIKTVYYVRQRLQTIEECVSCSI
>lcl|NZ_CP026520.1_prot_WP_042230762.1_2021 [gene=nrdF] [locus_tag=PC41400_RS10380] [protein=class 1b ribonucleoside-diphosphate reductase subunit beta] [protein_id=WP_042230762.1] [location=2260508..2261473] [gbkey=CDS]
MFDLKAFEAVNWNRSQHELTKVFWDQQWKQIWFPEEIAVSKDIKQWQDFAHQDTYKKVLGGLTLLDTIQTNIGMNEIARF
EPNLQEKALYTVFASFEAIHAKSYSYIFTTLCTNSEIDAIFDWVKRNEYLQYKAKRISDVYLSIKEGDEVTLWKAMFASV
MLESFLFYSGFFFPLYLGGQGTLRNSAEVISLILRDESIHGVAIGFFAQQRFKQFTAEQKEQLTVWAYEFLLDLYHNEVS
YTNDLYAETGLSPDVKSYVRYNANKALMNIGFEVMFPDEEVNPIVMNGIRNEGSTYDFFSQKGSTYAVAKVTPITDDTFR
F
>lcl|NZ_CP026520.1_prot_PC41400_RS30255_2022 [locus_tag=PC41400_RS30255] [protein=hypothetical protein] [pseudo=true] [partial=5',3'] [location=<2261539..>2261637] [gbkey=CDS]
RLDYDEVRKLAVKHKSRMLVAGASAYPRTINFE
>lcl|NZ_CP026520.1_prot_WP_042230760.1_2023 [locus_tag=PC41400_RS10390] [protein=hypothetical protein] [protein_id=WP_042230760.1] [location=2261894..2262085] [gbkey=CDS]
MDHTQQQRLLDFLKQEFSPDDTIHLLAISAVEQLHEEYGLVTTAPPCLYADSRLEEQLNTYST
>lcl|NZ_CP026520.1_prot_WP_053228726.1_2024 [locus_tag=PC41400_RS10395] [protein=hypothetical protein] [protein_id=WP_053228726.1] [location=2262619..2264085] [gbkey=CDS]
MSHSLKLLSWNIEKFNADKMNDDFFIKYVARVIRLMEIDVAGIMEIVGWIGNEVCTAIVNELNQQEQQKATGVVWSGQAS
EMTPSAPNEQYVFVWKLGLFSSVEWKLWNVLGENAFDTFFATNNITDEDEKFQVTQSLAKNGWMNDSFMVPYSAYSKLVA
DPQNLDCTKKNPTLNLTAQQKLDLSAILTAYAPETFPYPRSRPPFILRVTTADTNTVITFVLFHAPGPSDSSPITASNQI
SLIDPVIQADVGVVMGDFNVTEAWAKRTWALQYYNWDLGRLRYVLNDSGQYIFAAPFQRLSGPDFASHSTTQPALTTVQN
YSFRLQDNLTSMTSALVAPDAIADSTVMQSILTSPYDKFFVRSAQTDPSDPFVCNLVDLTTPKEIWTGTMDTDNNPVNET
RSTTPSTYQRDLSSLTAAIYNNWYNRHAGKSTSLNLQQIPKLNVRPTSLREAQYAYYYAISDHMPICMELKYEADPHYVM
SAEGVDFE
>lcl|NZ_CP026520.1_prot_WP_042230758.1_2025 [locus_tag=PC41400_RS10400] [protein=hypothetical protein] [protein_id=WP_042230758.1] [location=2264082..2268653] [gbkey=CDS]
MSLLDTLLGMYDAGTQQLNLNMTALGAWTAPLAQSSGSGAWQFSGVAQSPSEQAGTIIFAASGLKLTGTTPLILFPAQQF
SSIQCAIYNAETSPVFVVSAFASSNWNLANSFPGLQGTFWEELAVKTDDASRLPRLLLASGTVVEPSLNRSLSAGVSFFA
VFDETQGLFAPASWIVGPAPVGFGAIVPAAGLGTSLDIGLEISTSTNGFPGIFNNLQIPLAASLSFDPADQDATPGLRFS
AQLSWGGKQQTLSAVLPSESGGTVEIGLTGQLALPGQAELADYFGGTSITSYLPDPFKKAMNSICISGLYLGVGLGSKKL
EYAKVTISAFTDTSLDIIEDYIEIENIAVSVTVNDPFGSPSAAYRMTGGLDIGGIELDVAADLVPTIMSQVLSVSVDLAQ
GEQLNLTAIAEKFGLPSGAPSVMLTEFHMGGSTDGTFYVGASLEEIPTGSGQQAFVLSDLIFELDYSSQAEPATTFSMTG
SMTIAGVDAVLSLTAGNGWEIQGQTGMDTPIPIGTLIEDFGNRYGLTFPNPVSSLELQNVAFTYAKTAGAAQGSFSFSCE
GTFTIDTAPVSVAVTIDMEPDATGNGYHSTFGGTITIGSLTFALIFDTDNMQSDIFTATFSASGDLPSSISIYELVYALS
PEAASVVPQDLEVELKDVKFIYYKDASGSKYALGLDLSASFSLSDLPLIGSDMSGTDAFKVNDLQFLYASQPFAAAEAGA
VNALLPQGVYPLPSAGFGSGMTVGTELQWGGQTQTFMLAMGSTSATRTPAMRSYGAVGSVSDSTTSASAQWLSIQRSFGP
VALNRIGVSYQNGLLWFLLDGSLSAFGLTITLSGMGFGSPLSAFEPQFSLLGLGFDLKEGPLEIGGAFSEVSPAPDGTAF
SYEGGVVIETERFSIEGIGSYAQLTDGQPSMFVFATASAPLGGPPALFVTGLSAGFGYNSSMTMPAPEQVADFPLMALAG
GGTPPPLLGLLAELDGTADYAPGKRQAWIVPSSGDSWLAAGISFQSFGLIQSQALLLGKFGSDLEFALLGTSMLRLPKSG
DTVYAQAELELQAIWKPSSGFFGLTALLTPDSFVLDPSCHLTGGFAFYLWYDGPNAGQFVATLGGYHPAFQPPSYYPSVP
RLGINWALSDQFQIEGDAYFAITPGAIMTGGALKALYHDGNLKAWFTANADMLITFNPFSFQADISVSIGAFYRINLLFT
SKTISAELGASLELWGPPTGGKARVHWWIISFTIDFGADANAAQNDPLAWEQFKPLLPALDSVVTIKAAGGLAQSPAAGT
NGSGSKLDDSDLGWLVRAGTFSFTTQTAFPASHYAAGSSLAAIPDPAYPNGYSVQVRPMNTTDVSALHQIVIEQGGSEID
LSGWTLAPYGSNLPSALWGEPLKDAAGQFVLNPQVPNADTVANVPSGLAVSPPQTELGASRSALSGELLAFEDLSPQGEM
PLALSVEAEPLYVPVFAEDTPDSIAAIMGTAMQNRNSIYEVLSGLQIYTGSNGSLAGLASQADSLYCDSPLQVAAAGRSR
RDG
>lcl|NZ_CP026520.1_prot_WP_053228725.1_2026 [locus_tag=PC41400_RS10405] [protein=hypothetical protein] [protein_id=WP_053228725.1] [location=2268646..2271024] [gbkey=CDS]
MADSLPIGDIRFYDNKIPGLEAGTYMVTVSHQIEGLGLSAAEQQQLTAVQSFTVSGPQFALDASEIHSVVPPATASGDFG
GELPHIVLNEPILPWERSMSAVGTPWLGLLVFKEGELLPGDPDATDAATCAIRTTAGDFMALQGSGDILIPPGLVKDPAV
PDSLPCRYIQMSKEQFQAMTPYAAELPYLAHVREINAGDKASAGDDGWYGVVVANRLVASDQDGGNYTVHLVSFEGMESM
MAAEATIAQPAVALLSMTSWSFRCLSSDGENFGGLAAGITQGGMSGGEYNPDLLWLRHPQAGQVTAGDAAHTEAANRLMN
GYVALPYHTRTGEDTFAWYRGPLAPAVPATLAKSTPFLSSDAALIYDAGNGLFDCSLASAWEIGRFMALADESFGRTLLQ
YRQQSTGLTQKLYAHAEAHGLTEAGDLQKLAHPGAVKHSWLDTLAKDLGRLIGRKHSAESAESEQASAPAASGAVRKRAE
SPSSDRTASLRRFMQQEDIRQMARDGASEPLQAAAAWIARKVLLENVPFGHLVPIESMLPPESIRFFYMDMNWMDAMLDG
MMSVAMQSRLDTLAYQLDKEAIVAAVKSNLESYGIGLQGDGADPGPDGETRVAGFLLRSALVSGWPGMSVRAFDAQGNPL
KLLRLSRLSSGIMLALFWGVPSSLAFSEPQEGLRFGIDDAGSLDLRSLSTTNAQYPFGSELGIHLPLSGDYLRNGEYLNM
NPDDSNGLVQSVAGLISQKLAEGALAISPSDLALQLVKAPERQIFVPDSSAFALPENGKQERETGRNQGGND
>lcl|NZ_CP026520.1_prot_WP_042230756.1_2027 [locus_tag=PC41400_RS10410] [protein=hypothetical protein] [protein_id=WP_042230756.1] [location=2271024..2274839] [gbkey=CDS]
MADPIVLVPMEVSALSANAYVLQNQPFRRWEETYGNIAQFDSAEPGPTDVTGGFGSSSTGVYLHWKLPDAWTHGCQQPDS
QQVEYPLTPNRWLIVRLSGPLAARTASAFVVESDAPGTADSSVWLFDPDIVSAWEGSADPLRQQAGSALSGAGPFITGML
GRSFQLADWKEQGIEQLFVTAVAPGNHLFNAYQSHCRQVFSFYDALADVTEPDSTLSYLVAGWYSNSASDPVASCLNGQT
GDEAEQLLSALLSDYGWSLDRAEGDPLPTGTVCHGMVYGVDWQPQGTMQTVKDRFTAAGGSISVAVGNTSIDAFAAMIGQ
QLQSLSAEGNAEAAQILAEQPNAAALLEAFQYNLLHLADSGDWAGLDYRIRQEWFGSKAGGYRWTIVDIAASDSGTGEGV
HDESWLTGLNADQAAFDAARLQLHDAQWELYATWWKEGRFATYPGFAQPSGVTQDQFAMALAAENDPSYNPDQPPGPANI
YSVPKKVLDLANLLSGTLAPKVPQPVYADAATTPDDAFQAGIAVFEQQLRAQGLLAQNRGLKAAAEPRFWQAQDPVVLIS
GAGNTAEIPQAGSQLPCRMTAQTIAGLQNGGTVIPASSIAGLPAFPASAGNLAEMLQGAYSSADFAAAVSAAAGALFTEW
LLCDPDNAASISVTYPAFTRERLAEHNPSDFEGGLPALLPEQWSQPWNPFMLKWVAQWYPLPSGGSNWTFQGTDYAYTGG
SQKLTPQAIGGDCLLTPQTAFLFKNRLMQAAAASGDPADASLGQLADLIGEAEQWDFISQRLSGFGDQLSLRDSRTNQAP
AADVAFTFDDGTTHTLAELTGNQFDSLPLILDTGGTTPFQAVRQGQFCFSYLALYDAFGQVLEVVGDSGLTSSSLFAPVV
AEGLKPDTPVVQANPQRFLQLPPRLLQPSRLDFRMVDANNPDQPTDLDSGANPVAGWLIANHLDGSVAIYGANGDAIGAV
SQKIGAHGAPILSLAFAPDGGYQSLAELTEASPQLAAMLQGLLAQSAAAFEDFLEAIDNTLWTVDPASVGLDESLGILSG
KPLALVRAGMQLSLADQPLGDTSWAATFYPPVPDFVVQPFEVRLGDSALREDGLVGYFLNGSFDSFYCVQPPAEADGYLI
AIGQGQYIPLRFDGKTSADLLMLVDPMSAVHAFTGILPAQPVKLDRRFFDTALDRLSVRLPVGPMLTAVGPNPQPISAPS
MPASAVKIPVPVERGSSWTWMELAAADEVTIQRLAVTDTDGRAVIDGSDRSLREGQMVVQLKKEEEKHERK
>lcl|NZ_CP026520.1_prot_WP_042230754.1_2028 [locus_tag=PC41400_RS10415] [protein=hypothetical protein] [protein_id=WP_042230754.1] [location=2274826..2276301] [gbkey=CDS]
MNENEYGAVTDAPVLLTYTIMTDPAPLSASTPTYNSYGTITVTVSPAAASVYCKEIDIYIPSGIVSQTDQFVASADSSNW
QTHVGKEVPGDRVGLNPRLSYVQVKFMCKSPEFYLFTSPVNLTVGIQQLTQQPGWIGSLGLTEMSTNDSSQPYILRSMTF
GVSVGAALLFARNFTASAADGGASPAGVFTNGQPIQLSWESNGTEFQIFEAAGSGPLYTGTDNSFKLSSGLAKTTTFILA
ASQTDLRAETQYIYDSVTVVISNEDSTPHSVTAAGTVSGALVESTGEMFAQGKFTGSGGASLEGQIDVNGNLTMTGPAHL
NGPLNVEGGIQCNGNALLGDTSINGTLSATNTAAFQSVIVSGLLGLRGGSNIFGVPMSLSSDNVAYIAHTDGYAIATITP
PAGDVSSTISTCIVTLQASIAGGIPYEVSATGGVATAYNSAWTTVNWYSPNSITLPVPQGYKWKITVNQYSKNDKNPTVS
LFWIPLGNNEG
>lcl|NZ_CP026520.1_prot_WP_042230752.1_2029 [locus_tag=PC41400_RS10420] [protein=TetR/AcrR family transcriptional regulator] [protein_id=WP_042230752.1] [location=2276854..2277444] [gbkey=CDS]
MTKSTDHGEQTKRNIAAKAKELFELKGYAATTMDEIRQLSGSSKGSIYHHFKNKEVLFLYILELSMLEWIGKWREMEKPL
GTAREKLYRLAEHYAEDFRNPLLRAAEEFSGSENGDHEIRNKLLELTRMHYPIFQQLLEEGMQSGEFRPLPLEDLTFILF
GLFGGLSVSYYNYDTDRIIGLYRNAIDVLLQGIEKH
>lcl|NZ_CP026520.1_prot_WP_042230750.1_2030 [locus_tag=PC41400_RS10425] [protein=MFS transporter] [protein_id=WP_042230750.1] [location=2277507..2278730] [gbkey=CDS]
MKALFANRVFRIVTFADLLQQLAIWIRNISLLFYVMEQTGSDPVAVSLLNVMEYAPIFLFSFIGGAFADRWNPKRTMIAG
DLLSALSIGAILLFVLGGSWQAVFAATVVSAIVSQFSQPSSAVMFKRHVPEELLAPAIGISQGIMSLFIIVGPVVGTLIY
SSLGLAYSLAVMIGLFLTAAAVQFSLPPSPRSPERGAVPILEDVKEGFRYIRSHRNLIVIALTFACIGLGSGLIQPLDIY
IVTERLQLDKESVQWFYALSGAGMLVGGAAAAILGPKINPRVVIFAGIAFLSFSIIVEALSVWVYLTASMRFMVGVMTAF
MQIVLGAIMMKLVEEAYIGRVNGTITPLLMAGTLTGSALAGPLMKSITLVPVYGLSAAIILIAAFASLGMKLEEAAAKVD
QSGAHSD
>lcl|NZ_CP026520.1_prot_WP_042230748.1_2031 [locus_tag=PC41400_RS10430] [protein=SDR family oxidoreductase] [protein_id=WP_042230748.1] [location=complement(2279032..2279826)] [gbkey=CDS]
MQLHLQGKIALVTGSTAGIGKAIATSLAAEGATVLINGRSEEKVTNTVHEIQDRHAGAKLESVVADLGTEQGCQQVIEKF
ADVDILINNLGIFEAAEYFDIPDEDWFRFFEVNIMSGVRLSRHYLNRMIEKKEGRVIFIASEAAIMPSQEMAHYSATKTM
QLSISRSLAELTEGTNVTVNTVMPGSTLTEGVESWLNTLYPNEKMTAEEAEKRFMRENRPTSIIQRLIRPEEIANLVTFL
SSPLSSAINGSAYRIDGGLVRSVF
>lcl|NZ_CP026520.1_prot_PC41400_RS10435_2032 [locus_tag=PC41400_RS10435] [protein=helix-turn-helix domain-containing protein] [pseudo=true] [partial=3'] [location=complement(<2280125..2280433)] [gbkey=CDS]
MIKYNIPVEATLEVIGGKWKVVILCYLTKGTRRTGQLKRLMPGITQKMLTQQLRELENDGIIRRVIYNQVPPRVEYSLTE
YGQTLSGTLDALCKWGERHIDNN
>lcl|NZ_CP026520.1_prot_WP_042230746.1_2033 [locus_tag=PC41400_RS10440] [protein=TIGR03571 family LLM class oxidoreductase] [protein_id=WP_042230746.1] [location=complement(2280450..2281409)] [gbkey=CDS]
MFEHHQTYQRTFKEGDLTLGLHIPLENYGMETPTMARQIELAQAAETYGFTTLWLRDVILEDPNFGDPAVGQIYDSLIYG
THLLSQTKQIALGTSALVLPLRHPLRLAKEVATIEALFPERLIMGVSSGDRRADFKGLGVDHPSRGVQFREALSYLEKAL
YESFPSINSRYGDIEQATLVPNLNRRIPTMITGFAQQDMDWIGMNGDGWMYYPQAPGRQVDAIHAWRESAARQNQSAFRP
FSMPMHLDLAENPDEHATPIRLGFHIGRNKLIELLNLYKSIGVNHLFFALFDSLRPADEVIHELGEYVLPHFPPHEIRV
>lcl|NZ_CP026520.1_prot_PC41400_RS10445_2034 [locus_tag=PC41400_RS10445] [protein=MFS sugar transporter] [pseudo=true] [partial=5'] [location=complement(2281426..>2281836)] [gbkey=CDS]
VAFGNSIGGKLANKNPIRALFWMFVIQAAVLILLSFLAPFKVAGVIGVILMGLLAFMNVPGLQLYIVQLAEKYVPSAVDV
ASALNIAAFNIGIALGSVIGGIVIGSMGLVHTPWIGALMVILAILLTGISAKLEKK
>lcl|NZ_CP026520.1_prot_WP_042230745.1_2035 [locus_tag=PC41400_RS10450] [protein=aldo/keto reductase] [protein_id=WP_042230745.1] [location=2282221..2283063] [gbkey=CDS]
MIKNLQDTVTLHNGVKMPGFGLGVFKVEEGPELINAVKVAIKHGYRSIDTAAIYGNEEGVGQGIREGLQEAGISREEVFV
TSKVWNADLGYESTIAAYETSLKKLGLEYLDLYLIHWPVEGKYKEAWRALETLYKEGRVKAIGVSNFQIHHLENLLKDAE
IKPVINQVECHPRLTQKELQAFGEKHGIQLEAWSPLMQGELLDNEVLTAIAAKYKKSVAQVILRWDVQNGIITIPKSTKE
HRIVENAAIFDFELTAEDMKLIDELNENHRVGPDPDNFDF
>lcl|NZ_CP026520.1_prot_WP_042230742.1_2036 [locus_tag=PC41400_RS10455] [protein=aldo/keto reductase] [protein_id=WP_042230742.1] [location=2283366..2284376] [gbkey=CDS]
MKNILQGKLGFGTAPLGNMYRTISEEEAAATVDAAWESGIRYFDTAPSYGAGLAEIRLGEALSTKNRNDYVISTKVGRII
SDELEDPSARDFGEKTGLFEFGCKNKIINDYSADATLRSIEQSLKRLKTDRLDIVYIHDVAQDFYGDEWLSQFETARTGA
FRVLTRLREEGVIKGWGLGVNRVEPIELMLDLEEAKPDVSLLAGRYSLLDHERALQRVMPAAVKHNMDIVAGGPYSSGVL
AGGTHFEYQKASPEIIAKVEKIKNIADRHQISIKAAALQFSLANPAVAAVVPSATRPERIAEDKAALNTVIPAAFWEEMR
EQQLVAPNAPLPINVK
>lcl|NZ_CP026520.1_prot_WP_053228723.1_2037 [locus_tag=PC41400_RS10460] [protein=leucine-rich repeat domain-containing protein] [protein_id=WP_053228723.1] [location=complement(2284490..2286211)] [gbkey=CDS]
MESLKKIGPADWQGITTLNGLEYAKNLVEIKLAQNKIENIDALSGLTDLKTINLWQNQISDIHALSNLRSIVNLNLSQNH
ISDISSLRNLNSLKVLNLSENEIQNMIPLENLENLSAFNAYKNQIQDVSPMRNFKNLASTNLQENQISDISPLSNLSKLD
FIGLKYNRVKDINPLKSLTHLTGLELTGNPVQDLSVIKHFPKLTLLSIGSLHISNIDFLENHPDLKWLQLENNQITDISS
LQKLGKLQDLDLSNNRISDVSPLSHANKLELLKLNHNRISDIKPIVQLPNLWLLSLNGNSISDPESLGSLPQLRSLYLGS
NGIASLQFLKSLPRLNLLSLDGNLISDLRGIEAQNGIYELDLSNNRLTDLSPIKSLKMLDVLFLDGNALSDISALSQLTP
RKISLVNNQILDISPLDNQTDIWEIYFANNPLNEESVSKLKVRALNGAAVSYGERIFVKINNEVQNFSEDESPQNISGSI
HVPMRKIFEALGANVTWDSNSETVTAQKLDISLKLIIGSNKAIVNGKDIFLESAIGLVNGSVFVPARMVSETFGAKVNWD
SHTKTVDIRQDSD
>lcl|NZ_CP026520.1_prot_WP_129112471.1_2038 [locus_tag=PC41400_RS10465] [protein=hypothetical protein] [protein_id=WP_129112471.1] [location=complement(2286223..2286444)] [gbkey=CDS]
MEIFQQVKENDVLITGGNYLKKKTRFVIFSFSMVVALVLLLPTKTFAESSKVKIEDSALKKSLEMNSIKMKAQ
>lcl|NZ_CP026520.1_prot_WP_042230740.1_2039 [locus_tag=PC41400_RS10470] [protein=hypothetical protein] [protein_id=WP_042230740.1] [location=2286599..2286889] [gbkey=CDS]
MKAKRIDFDGNGIQGLKEIWLSILLGENKAANGPLRYDWEGIQNIDERLKWLDNNEVESIVVRRALLADPAWARRIHEGR
VVEIRAFTPDSLHTLF
>lcl|NZ_CP026520.1_prot_WP_042228795.1_2040 [locus_tag=PC41400_RS10490] [protein=AraC family transcriptional regulator] [protein_id=WP_042228795.1] [location=complement(2292584..2293447)] [gbkey=CDS]
MRLDNYADLGRHPVQLSLYLNRSESFRDIFHAHQGIEILYVHRGNGRVMIGQELFEMKPGTLLFFKPFQLHTIRMTITEQ
QPYIRSMLLFDPAVLLRYTEPYPSLNSFLLHIWKHPFGDQWVDVTDTDAMDKLFHFYKDESQVSHNAPVQSEKVLLDTLS
LLHRIQNLFDKQSAVGRVGSSLRFSPSVTAILAWIEANYASEFRLTDLARHVHLTPNHVSYLFHRETGSTLTNYLTAKRL
QQASFLLKATNLSIREIGHRIGFSNFSYFCQLFKKHNGLSPAKYRTS
>lcl|NZ_CP026520.1_prot_WP_042228793.1_2041 [locus_tag=PC41400_RS10495] [protein=glycoside hydrolase family 88 protein] [protein_id=WP_042228793.1] [location=2293565..2294704] [gbkey=CDS]
MEWYAGRIAGKAGITVNKEELWGSLHKKVNVMAEKMGNKSPHVAFDGVYDDMRIDWWTSGFWPGMLWIMHEMTGDPSFRE
KAWDWDASLEQCMVSNSNLHHDVGFQFLPTAVIKYKLTGDADARRRGIQAANFLAGRFNLAGRFLRAWNQDKTGWAIIDS
CMNLSLLFWAAEEIQDPRFRHIAAAHADTVLEHFIREDGSVRHIVSFDPETGAFLETLGGQGHSPESGWSRGQTWALYGM
ANVYRYTGDIRYLHAAKRVAHYFLASLPEDAVPHWDFRTGNSEGEPRDTSAAACAASGLLEIAKHVAAPERELYDRAALR
IMAALAGSYASFDDKTEAILREGTGNKPAGQNIDVGLIYGDYFLVEAAAKLNGWQGSIF
>lcl|NZ_CP026520.1_prot_WP_042228792.1_2042 [locus_tag=PC41400_RS10500] [protein=hypothetical protein] [protein_id=WP_042228792.1] [location=2295121..2295399] [gbkey=CDS]
MTTKTLTVPQRQKNEREVGRLFSFMNLMAWSSLVIGIIMSLWNVFHFSDKNLTLMVGIGFMVGSVFIYTIGTAIHLVHER
KFEQQENGEVTE
>lcl|NZ_CP026520.1_prot_WP_042228791.1_2043 [gene=cyoA] [locus_tag=PC41400_RS10505] [protein=ubiquinol oxidase subunit II] [protein_id=WP_042228791.1] [location=2295844..2296854] [gbkey=CDS]
MKKGNLFRRFSAVFMLVLMTVVSAGCKSPIVLDPKGPVGQQQMDLIVISTLLCLVIIVPVLVITFVIVWRYRDKPDRKAK
YTPNWEHSTKLETIWWGIPILIILVLGVVTVRYTHALEPSKPLAAEKEVKPITIQVTSLDWKWLFKYPEQGIATVNYVQF
PEDVPVRFELTSDAPMNSFWIPQLGGQIYTMSGMAMKLNLMADEPGEYMGSGANFSGKDFGKMRFVAKATTQADFDSWVK
EIKGSSPELTLDGYKKLAEPGVSEVQSFSKFPEGLFEKIVTRYSPSHNHGGFGAPVPAKPSAGTESGTSDKAAGNSSTEG
SGAVKQEASAAHTGHE
>lcl|NZ_CP026520.1_prot_WP_042228790.1_2044 [locus_tag=PC41400_RS10510] [protein=cbb3-type cytochrome c oxidase subunit I] [protein_id=WP_042228790.1] [location=2296878..2298848] [gbkey=CDS]
MLDKIKDFASTFFVTGDPLIYGADVSIALATIAIIFVLTYFKKWTWLWREWLTTVDHKRVGIMYIIASILMLFRGGVDAL
LMRTQLAMPDMEFLQPEHYNQIFTTHGVIMILFMAMPLMFGLFNIAVPLQIGARDVAFPFLNALSFWLFFFGAMLFNMSF
VIGGSPDAGWLAYPPLSELSHSPGVGQNFYIWGIQISGIGSLMTGINFIVTILKMRAPGMKLMKMPMFSWSVLSSCIAII
FAFPILTVTLALLFLDRFMGAHFFTLDGGGNPMMYINLIWMWGHPEVYIVVLPAFGIFSEVVATFAKKKLFGYKSMVFAM
MIISILSFFTWAHHFFTMGSGANVNAFFAITTMIIAIPTGVKVFNWLFTMYRGKIRFDTPMLWTIGFIPCFIVGGMTGVL
LSVAPADFQFHNSYFLIAHFHQVLIGGVAFGYFAGLYYWWPKIFGFKLNEKLGKWAFWLWNIGFYVCFMPQYFLGLDGMT
RRLSTYGWDSGWYGLNLTSTIGGFLMGIGFIFQVWQIAHSIKFMERDTTGDPWNGRTLEWSIPSPAPLYNFATIPTASVQ
DPWWEEKQRLEQGGKPAPKAKLEPIHMPKNSAIPFIKSLCWFIAGFGFVFGWMFFAIPGMIGVAVCMLAHSFNYDTDYYI
PVDEIERTEAAARGSV
>lcl|NZ_CP026520.1_prot_WP_174446798.1_2045 [gene=cyoC] [locus_tag=PC41400_RS10515] [protein=cytochrome o ubiquinol oxidase subunit III] [protein_id=WP_174446798.1] [location=2298879..2299478] [gbkey=CDS]
MSHQHDHSHDDGHHDLESLRMFGFWIFLITDVILFGTLFATYIVLHGNTNGGPTPAELFEMPGIIAETFILLTSSFTSGL
AVLAMNKGNRKGLIGWLAVTALLGASFITLEVTEFVKMVHEGATISTSGFLSGFFTLVGTHGIHVSVGLVWMIALIIQLS
RHGINSVTRRKVNIISLYWHFLDVVWIFVFTFVYLMGVM
>lcl|NZ_CP026520.1_prot_WP_042228787.1_2046 [gene=cyoD] [locus_tag=PC41400_RS10520] [protein=cytochrome o ubiquinol oxidase subunit IV] [protein_id=WP_042228787.1] [location=2299480..2299824] [gbkey=CDS]
MAQHTTHSGGAHDHEHHESHGSLKSYVIGFLLSIVLTIIPLVVILNKMVTGKAAILVLLITAVLQFAVQLLFFMHLTDEK
GPRYNLLTLILGLVILTTIVAGSIWIMTYNTVAN
>lcl|NZ_CP026520.1_prot_WP_129112472.1_2047 [locus_tag=PC41400_RS10525] [protein=hypothetical protein] [protein_id=WP_129112472.1] [location=complement(2300744..2301064)] [gbkey=CDS]
MKSRVLAMLFLCFALFIVVPVSASAATSVQVTGQGQLAISASFTGASYAEVYVSNQNGQAVYNKIFASVGGNGTISASIP
NLPYGTYKIIIGGQAISVSNLQYGFQ
>lcl|NZ_CP026520.1_prot_WP_042228784.1_2048 [gene=argS] [locus_tag=PC41400_RS10530] [protein=arginine--tRNA ligase] [protein_id=WP_042228784.1] [location=2301630..2303546] [gbkey=CDS]
MISATLGIEIERGVRKLLDEAGTSWPSELKVLVDRPAHPDHGEYSTNVSMLLAKTLRKSPMEIANKLKSELEHAKIAEGI
FSKIEAVAPGFLNFAIDWNVWLNRHREYSAYQGETEKGKVVIEHTSINPNKSAHIGHLRNSCIGDTLARLLSRAGREVEI
HNYIDDLGNQLADTVVGILHTKTEGEHVRFGDFCWDTYAKVNKVYKGSPELQEERTRVLHELEEGTSSTAWIGRLTAEKI
VREQLHEMKAFGIGYDVLVWESRIVAEGLWEAAFEHLQATGAFYKVEEGKLAGCWVLRHGGEGSGEAEGTPSPDPEAHQA
DKVLVRSNGILTYTAKDIAYHLWKFGLLSQDFTYMPFADGVWSTDSQGQAMPFGRGETVINVIDYRQEYPQAMVKLALEA
LGYKEQAARLLHVSYGVVSLSPETAESLGVDTSAGKGSYAMSGRQGVGIKVAELLQVMEESIESKRSRQEGLSSREIAAA
AIRYYLLKYHLQTEVVFDLRQATEVSGNSGVYLLYAYARAGSVLEKAGAGQKGSGKALMEPPVPSFDGGALEPPERQLLK
QIAYWTETMDAAVRQLAPNLICSYAFELSLLFNNFYGACPILKGPDDRVELRVWLTWKFRETLGKALDVLGLPAPDRM
>lcl|NZ_CP026520.1_prot_WP_042228782.1_2049 [locus_tag=PC41400_RS10535] [protein=D-alanyl-D-alanine carboxypeptidase family protein] [protein_id=WP_042228782.1] [location=complement(2303693..2304631)] [gbkey=CDS]
MKLKSTLLGLTAAVVALALVSDPLHIIKPRAVSALSTLRSGLLGGSGSPLTSSVEGEAAVLMDADTGKVLFEKQQDSRLY
PASTTKIVSAMIALEKGGAKDLITVGDEIRLKTPGESTAGLWEGQRLSLEDLIAAMMLPSGNDAARTVARYIAEKDSGRQ
LSAEESIDYFAGLMNKKAAAVGAKRSHFTNPHGLHDPKHYTTAGDMALLAREAMKNKVFRGIVDQPRHTVASAGNKQTFV
NRNQLLQEGGAYYFKGVNGVKTGFTDEAGYCLVSSADLDGKNLITVVLRSSSGGVYTDTHALLKLAVNKETA
>lcl|NZ_CP026520.1_prot_WP_042228780.1_2050 [locus_tag=PC41400_RS10540] [protein=HAMP domain-containing sensor histidine kinase] [protein_id=WP_042228780.1] [location=complement(2304823..2305959)] [gbkey=CDS]
MKTNWSRALLYSLGSSLIVLLLIFLLGRTLFYSYLSERISLENILIWINYRIGYPDAYIQAGLIIFLLFFIYFFRLEQRH
RDKRTLAQIIENMQKITDGDFNHRVPAGGSGELGQLAVHVNRLVDRLKLSLEEERQAEQAKNDLITNVSHDLRTPLTSIT
GYLGLVDQDRYRDEVELRHYIGMAYEESQRLNQLVQDLFEYTRLRNKEMKLQLVKIDLVEMLHQIFSAFQLQLKEADMDT
RMTFEQPRLYVLADGNKLRRVFENLITNAIKYGHDGAFLDVTGRVEGSEIVLSLTNYGESIPAPDLPRIFERFYRVEKSR
TKHLGGSGIGLAIAKQIIDLHGGTIEAYSDPEETVFTVRLRALNPDNPPPSSVRSGSF
>lcl|NZ_CP026520.1_prot_WP_042228779.1_2051 [locus_tag=PC41400_RS10545] [protein=response regulator transcription factor] [protein_id=WP_042228779.1] [location=complement(2305949..2306644)] [gbkey=CDS]
MDKHILIVDDDDKIAQLIEIYLQNEGYTVYKANNGLTALDIMAEQTVHLVILDIMMPGMDGLQVCMKIREKQMTPILMIS
AKDGDLDKITGLMTGADDYVVKPFNPLELIARVKSLLRRAYFQAQASVPAEEHTTRIGSLLIDKQTHTVTANGRLVKLTP
IEFGILSLLATHPGRVFSSEEIFEKIWKEKYFESTNSVTVHISRLRDKLEKEMEGEKIIHTIWGVGYKLEN
>lcl|NZ_CP026520.1_prot_WP_042228778.1_2052 [gene=msrA] [locus_tag=PC41400_RS10550] [protein=peptide-methionine (S)-S-oxide reductase MsrA] [protein_id=WP_042228778.1] [location=complement(2306829..2307797)] [gbkey=CDS]
MSLENKSSELATFAGGCFWCMVKPFDEQPGILKVESGYTGGHLENPTYEQVCSDTTGHYEAVQITFDPAVFSYERLLQVF
WQQIDPTDAGGQFNDRGQSYQTAVFYHSEEQKAAAEKSKLDLARSGRFDKPIATEILPAKPFYPAEEYHQGYYKKNPLRY
KAYSVGSGRAAFLKNTWKDHAKDQDLKKRLTPMQYEVTQNSGTEPPFRNEFWDHKEEGLYVDIVSGEPLFSSLDKFDSSC
GWPSFTQPLQGEQVTEHVDLTHNMVRTEVRSKDADSHLGHVFEDGPGPNGLRYCINSAALRFVPKSKLEEEGYGDYLKLF
TK
>lcl|NZ_CP026520.1_prot_WP_042228776.1_2053 [locus_tag=PC41400_RS10555] [protein=NUDIX hydrolase] [protein_id=WP_042228776.1] [location=2308029..2308586] [gbkey=CDS]
MEANYCMTCGTPLETRDMDGTLRRACPACSFVHWGSYSAGVGALVMREGKLLLVRRAQEPGKGRWTNPGGFIEQLELIHE
TIAREVLEETGIEASVKKVVAFRDMPKSIHNIYIAFAMDYVSGEPVPDDHEVDAAGFYSLEEMETMNVAPFTRWLADVAW
NGRGGGLVIDTDPVIPLDGYGLFRV
>lcl|NZ_CP026520.1_prot_WP_241688837.1_2054 [locus_tag=PC41400_RS10560] [protein=glycosyl hydrolase family 18 protein] [protein_id=WP_241688837.1] [location=2309007..2310524] [gbkey=CDS]
MKKWKPSLLTALTLGLALGAVPAMGASSAGDISIAYGSRVAVQQQKPVLQGDALTVPAEELAKQLQLKLSPLKEDGTFTL
SKTGVFIEMKLGSKEARVNNRLATLQAAPAKQGDALYVPLRFVSEASRSLVEWDQARQMAIISPAYKVVGYFTSWGKLKA
SEIDASRLTHINYAFANIVDGEVALSSPKEQEEAYPGACKGEDCAKFVSDFKELAKLKQANPHLQTLVSVGGWTWSGRFS
DAALTEASRTKFADSAVRFIRENGFDGVDLDWEYPVTGGLETNVRRPEDRTNFTLLLQTIREKLEAAGREDGRHYLLTIA
SGANPNYVRDTELDKVSESIDFVNIMTYDLHGSWEPTSNNNAALYYDPADPDPNAHRFYVEAAVNGHLNAGVPAKKLVLG
IPFYGRSWGGCEAVNNGLYQACKGGDVVDWTEKPEGFQRYWNDRAKVPYLYNKQTGQFVSYEDEESIGYRAEFIKEKGLG
GAMFWELNDDRDSKLLPRLAESLDR
>lcl|NZ_CP026520.1_prot_WP_042228774.1_2055 [locus_tag=PC41400_RS10565] [protein=metallophosphoesterase] [protein_id=WP_042228774.1] [location=2310881..2312023] [gbkey=CDS]
MKSRLVLLVLALLVYAGLTIYIGWHGERFLDFWGISVSAGLYWTLLAVLALAYMLGRSPAVPGPPGRLLKVIGSYYFAVL
EFAVILLPLGDLAAWLLKLAGFPPAVYFGGTGGVVLVILIILFAKGYWNAWTPIVRTYRARVGKGAGPLRQLRVAVASDI
HLGNIVGKRHLARLVKTVDEMKPDLILLPGDLIDDSIEPYIRNGMSEVMKQLQAKYGVYAVLGNHEYYGGHIEEHISQME
SIGIRVLRDETVSVEDAFYIAGRKDKTAESFDTGGRMAVDRLLSGLDPSKPILLMDHQPYHFDKAAAAGADVLLCGHTHR
GQFAPNHWITGRLFELDWGYMLKDKMHVVVSSGFGTWGPPIRIASRSEVIELILDFEEAE
>lcl|NZ_CP026520.1_prot_WP_042228773.1_2056 [locus_tag=PC41400_RS10570] [protein=aldose 1-epimerase] [protein_id=WP_042228773.1] [location=2312090..2313082] [gbkey=CDS]
MTHNRAYESNYQGMKAVRLEWGAYEAVMLPDYGGNLIAFRDTDNGYRFLHEPAEEELESFKANPGIHGIPVLFPPNRYED
GKFPWNGKVLRFPINEESTGNHLHGFLHKIPWQVDEYGTTESESFVSVSVRVDEAHEVYKFLPFRFTIRLRYSLNGDGLA
QHLLIRNEGGERMPCLLAFHTSVNAPFAPDASAKDYRVKLTIGKRWEMNDRMLPTGRQQALTQEEEQLRDEGIYPFYEPL
DNHYTVSAQNGRNRMELTDTKHNVTLVYDVGTSYKQWMIWNHFGEEGFFCPEPQVNLVNAPNSPLPAEEIGLFGLNPGEI
WEETGRLYVK
>lcl|NZ_CP026520.1_prot_WP_042228772.1_2057 [locus_tag=PC41400_RS10575] [protein=hypothetical protein] [protein_id=WP_042228772.1] [location=2313173..2313859] [gbkey=CDS]
MTTPNESGDQPIEAQATEEEQLSPEEQAAREQQEQQQQNEIIFAWYKHAEQVLKEQFSDYEVEGQVGQHPVFGPLFAYTI
RQNDNSYSCGFFLNELVHTFQNKENPGLWMSSFFVDLIRSSDNRPLPAAPQSEEEAKQLLDNYIVPHCAESVRSEFPEEQ
IYVDLEMHPEAGPVLEVGFPSIKEGNNTFALPLQFLMTLHLLNRDPADPLVQALYKIHEEHQQQAEHT
>lcl|NZ_CP026520.1_prot_WP_042228770.1_2058 [locus_tag=PC41400_RS10580] [protein=helix-turn-helix domain-containing protein] [protein_id=WP_042228770.1] [location=complement(2313984..2314913)] [gbkey=CDS]
MVQAVVYSETVNRTLSYIEAHLKEELNLEILAERAAFSMYHFHRIFQAQVGMPPADYIRRRRLSSAACSLLGTERRILDI
ALEYRFDSQESFTRSFKRMFLMTPGQYRSYYRTLTMQKESEAEMAEQADQMNALQGEPRGWMLTGTNTGDYEMGVDRLEV
HHGKSSGYLRSLTERASGFGTMMQMFKADAYLGSRLRLSGFVKTEQVRDWCGLWLRIDGKDHDVLGFDNMSGRPIRGTTG
WSRYSLVLDVPKESVAIAFGVLLNGSGRVWIDSLQFDTVSLQVPVTSLEEVNALPDAPVNLNFEEEPDQ
>lcl|NZ_CP026520.1_prot_WP_042228769.1_2059 [locus_tag=PC41400_RS10585] [protein=polymer-forming cytoskeletal protein] [protein_id=WP_042228769.1] [location=complement(2314987..2315715)] [gbkey=CDS]
MEAQPIITNDLSISGAGSADGGRYRNARINGNGRIDGDLECRSFKSAGHSHVYGNLAAQEVILRGSGSVAGNVRSELLSV
SGNGEFRGDVAAGLVKVNGSAGIGGSLTADTAVIRGSVQINGDCEAEKFEAYGGFEVGGLLNAGMIDIRLFGPCRAREIG
GESIRVNKNGFLNSLASLFGSKRISLEAGTIEGDVIYLEHTFASVVRGRDVKIGPGCEIGRVEYTNTLYVDIDSTVAQRQ
QL
>lcl|NZ_CP026520.1_prot_WP_042228768.1_2060 [locus_tag=PC41400_RS10590] [protein=YhbD family protein] [protein_id=WP_042228768.1] [location=complement(2315718..2316350)] [gbkey=CDS]
MTQELISKKDLLELTGISYGQLYRWKRKNLIPEAWFIRKSTFTGQETFFPKKEILVRIDNILYQKDDLSLDELADRFSPR
PTDLSMTKEELILHNIVSKVALDYFIRLSGDAEVWTFERIFAVYVLHTLLETGEISLEEGSGLIQLMTDQYPNMSDSDGE
LLFIRKMGVSTCLLKQGTGEIYLEKGVKVAARLQLSPSVEALKTKIWREA
>lcl|NZ_CP026520.1_prot_WP_042228767.1_2061 [locus_tag=PC41400_RS10595] [protein=sulfonate ABC transporter substrate-binding protein] [protein_id=WP_042228767.1] [location=2316631..2317680] [gbkey=CDS]
MTTISGWLKRSTRTKAWLAALMLTAAAVLSACSSGGDKGAAATTPNEAKSAVTGDSKVLRIGYQKGNTLNILKEHGNLEK
RLKEKNITVEWKVFAGGNFLLEALTAGSIDFGHAADGSGVFAQAGGKPFVYVGCDLPNPEGMGIVVHAGSPVKTVADLKG
KKIAVAKGGNHHYLAVLALEKAGLKLDDVKFVFVKDASEGRALFETKQVDALGSWDPFFASVENDLAPVTLTDGTGFSPN
RTFYYANETFAKGNAELIKTILEEIDVSDKWANANKPEVVKQLTEALGIDRKAIERAVNRRMYGVENLSPEIIEPQQKLA
DTFHRIGLIGDKINVAPLMPVRQVWAIGK
>lcl|NZ_CP026520.1_prot_WP_042228765.1_2062 [locus_tag=PC41400_RS10600] [protein=ABC transporter permease subunit] [protein_id=WP_042228765.1] [location=2317712..2318515] [gbkey=CDS]
MSVSVEQAYTRRRDTWAAGLLPWLLPVLIVAGWHMAAVYGPVSQRLFPPPLEVAKSGWKLFASGQLTYHLSVSLGRAAAG
FVIGGGIGLLLGAANGLFVTSYRIFDTSIQMIRNIPHLALVPLVILWMGIDEGAKVFLVALGVLFPVYVNTFHGIRSVDP
GYLEMGRVYGLTSWQLFRHVILPGALPSVFVGIRYGLGIMWLTLIVAETIATDKGIGFLALNAREFMQADIIILSILLYA
LLGKLADVIAKWLEHTVLKWNVSYRKA
>lcl|NZ_CP026520.1_prot_WP_206735007.1_2063 [locus_tag=PC41400_RS10605] [protein=ATP-binding cassette domain-containing protein] [protein_id=WP_206735007.1] [location=2318541..2319383] [gbkey=CDS]
MSGWGQLAEEALYTEAPAGDKRRAVSGNDAGQSGESLFGASVRLQGVHKSFGGKKVLHELDLTIRPGQFIAVAGRSGSGK
STLLRLLAGLEKPAEGRIEIDGRPVSGIGRDITVMFQEARLLPWKTVLDNVGIGLSGNWRPRALHVLGQVGLTDRAGEWP
GVLSGGQKQRVALARALVRQPKLLLLDEPLSALDALTRLEMQQLIGTLRQRHGFTTVLVTHDVSEAVRLADRILLIEDGK
IALDADNPAPWPRDQANPEFARLEKRVLERIMQTGGGNGR
>lcl|NZ_CP026520.1_prot_WP_042228763.1_2064 [locus_tag=PC41400_RS10610] [protein=acyl-CoA dehydrogenase family protein] [protein_id=WP_042228763.1] [location=2319380..2320582] [gbkey=CDS]
MNPSPFSFARTSEEQKRLDEIGRLADTLRDEAARADEENGFSYGAIDALKEAGYPSLTVPAEYGGAGASLYEFVLLQERL
AQGDAATALGIGWHLGIVLELANQPAWQDKYEALCRDIASRNLLINRAATEKGTGSPSRGGRPATTAEEVEGGYLLNGRK
TFTTLAPALDLVIVTASLEPDGNHGEFVFPVATEGVRIEPTWDMVGMRGTASHDLVLEDVLVPPEARVKHTRSEPSRPGP
AAHPYLLHIPACYLGIALAARREAVTFAAAYRPNSLPHPILELPHIRQLIGEIELELSAARHFLYAVAQRWESAPANPGA
FAADLSAVKTFAVRTALTVVDKSMRIAGAHSLAMTHPLQRLYRDVRFGLHNPPMDDVTLKQLADRAERELLPPPEGTAEA
>lcl|NZ_CP026520.1_prot_WP_218002296.1_2065 [locus_tag=PC41400_RS10615] [protein=transposase] [protein_id=WP_218002296.1] [location=2321035..2321310] [gbkey=CDS]
MNRYVTEALQIVRKRVQKDLSPRARQQLKHHYRLLGKGADQLSEREAGLVKQFLQYSDILCAVYQWKEMFITWMCLKRMC
LKSVDKEIYIL
>lcl|NZ_CP026520.1_prot_WP_053228685.1_2066 [locus_tag=PC41400_RS10620] [protein=LacI family DNA-binding transcriptional regulator] [protein_id=WP_053228685.1] [location=2321364..2322485] [gbkey=CDS]
MPFNLKTIIDYFFISLFNRLNSYDSNIRRKREIEVEGKKPSIKDVAKKAEVSTATVSNVMNASRFVKEETKLKVLKAMEE
LNYRPSTVAKSLKGKDTKIIGLIIPIRQNDTSADFFISLANGIESVLNTKGYRLVISNSHENINNELEQIDMYNTQFTDY
MDGLIIAPTSQNGKNVNELSYLQYPVVYVDRKPSVQNKVDTIYTNNYQITYEAIQLMVQKGRKRIAFISGPIDVSSTIER
FQAYKDVLKDNQVPFDEKITYVGESLFESGYELTQKVLDQEIDGLVIVNNTMSMGAYKSFKEKKVSVPKQISFLSYDDFQ
WMSLTEPPITTIKQPAFEMGQAAAALMLEKLENLSKDPVEICIDSTLVIRGSL
>lcl|NZ_CP026520.1_prot_WP_042228761.1_2067 [locus_tag=PC41400_RS10625] [protein=extracellular solute-binding protein] [protein_id=WP_042228761.1] [location=2322516..2323805] [gbkey=CDS]
MKKFSYLLILLLVVSVVIAGCNSGGGDSNEDGITTIDFAIHVANPMEQEPAFSKLIEQFQIENPDIKVNLIGKEQQEHVK
NIKMMSQSNKLPDVFWMLPASASELYEAGMLMGLNDFIKENPEIKNSLNENMLSTYAANGEQYGLPYQPLVTGLFYNKAL
FKQHGVELPKTIEDLVAAAKVFKQNGVTTIAKGAKDEYSVWAFLTMLSRYGYFEKIDNIMAGKEKYNNEDFVNYYKKIDE
LRKAGAFPDNVSTQTYFQAVEQFLGGQAAFLDSGMWDIPKIEESGIDAGFWWGPTFADGVGNQNISSVVPAAPLLVSKNV
EKDAAKKEAVLKLFAFYYGEKGTQIMVENQVPPMTNTKVTIDEAKNPLFKQLVDQIQLKGWVSQPNQPDLVVPEAIGNAM
YDSIYGVINGTYTPEEAAKVVQNKVDETK
>lcl|NZ_CP026520.1_prot_WP_042228758.1_2068 [locus_tag=PC41400_RS10630] [protein=sugar ABC transporter permease] [protein_id=WP_042228758.1] [location=2323866..2324768] [gbkey=CDS]
MRWFSSNSNKIIMLLPTMLLYMVYIILPIIIAAYYSLTRFTGIGKPKFTGMSNYLRLFNDPIFMVALKNTFIVLVVTLII
LIPAAFLLSLLLNKKMKGLGVIRAINYSPSIIAPIMVGLIWVFILDPKIGLINNVLNMIGLESWALQWIGGDKLTPFSVA
VVQSWQSLGYIATIFLAGLTLIPKEMYEAADIDGATKFQQLRNITVPLLGETFKLNIILAITLCFKIFETVLQLTNGGPN
HLSDVLVTYMYSTTFNAGEYGYGMAIAVASLIATILLTGIILGMFTFGKRMVNSEGARKA
>lcl|NZ_CP026520.1_prot_WP_084706569.1_2069 [locus_tag=PC41400_RS10635] [protein=carbohydrate ABC transporter permease] [protein_id=WP_084706569.1] [location=2324774..2325643] [gbkey=CDS]
MISTAQGTKVAVSKRKSGRLLSRLLTFGITALVMIPFPWMILLSFKDNQSILTDPLSIPKSFSLENYVRAFQTLDMGLLY
KNTFFIAVVSVFMGLVVSFMSSFALSRMVFKSNKWKNGLYLFLIAGLAIPPFILLYPIYKITIFFGIENTYLSLILPYIA
TTLSFNTLLFVGFLKDFPSEIEEAALIDGANLYILCAKVVIPIVKPVMMTILIFNTLYVWNEFPFAATLITDQSLMTISM
AISEFKGRYNIDYGGIVSANLLFIIPQLIFFTIFQKHIVEGMTAGAVKG
>lcl|NZ_CP026520.1_prot_WP_042228757.1_2070 [locus_tag=PC41400_RS10640] [protein=DUF2961 domain-containing protein] [protein_id=WP_042228757.1] [location=2325682..2327025] [gbkey=CDS]
MTNLSNGSMQHLTTHKDVRSARVSSSDQTGRNQDYWLIPPNESVVLGEIEGPGCLTHFWMTSFCRKIMGPSIVDPALGSN
VAPVNEIHNALGVTWEEQDPFYYRKALIKITWDDQDTPSVLAPFGDFFCVGHSLPGNFQSAPFNVSVKPEEANKFGAPAA
LNCYFPMPFNKKAKIEIVNENELPFGLYFYIDYELYKEELPEDTVYFHANWHRENPCEGWGNDLQVNSPAVQSVANLSGE
ENYVVLETEGKGHYVGCNLSVNHFQGSWWGEGDDMIFIDGETEPSINGTGTEDYFGHAWGMQKNAFLYNGSIIHESEKPG
YQVSYRFHIADPIHFAEKIKVTIEHGHANHLSDDWSSTAYWYQTLPSPKLDILPVEQRLPHVAVMPEVKQDTKPELNDEQ
RKAYELAEKLEQSYVTEKKNQMEIKFNKTRESSVGNKEHVQRVKSTL
>lcl|NZ_CP026520.1_prot_WP_042228755.1_2071 [locus_tag=PC41400_RS10645] [protein=glycoside hydrolase family 32 protein] [protein_id=WP_042228755.1] [location=2327124..2328596] [gbkey=CDS]
MLTSKHQVALDAANKIIEQKKEHVAQSTYRLHYHFMAPTGWINDPNGFVQYSGEYHLFYQHYPYDSKWGPMHWGHAKSDD
LIHWKHLPVALAPSEPYDSGDVEGYGCFSGSAFVNNGELVLAYTGHVDGNTPQQVQCIATSKDGVHFEKYEHNPVIDQFP
SEGTHDFRDPKVWKHEDKWYMVVGTKKDGKGKAALYTSEDQHKWEFKGIAAESDGTQGDMWECPDLFPLNHRHVLIVSPM
YGKQNENPFYVIGDMDYEAGIFTQKVSSILDYGFDFYAPQTLVDDKGRRIVIGWMEKWLTKMPSQEHGWAGAMSIPRELV
LESNHVVKQKPIEEIVSLRSDYQKWDVSAIDGVKVISETYDSVSETIIEVDVNETDAAAFGLQLRCSQDGEERTDILFDL
KNNEIKMDRTRSGSGEKGVSTAPLHIKDGSIHIRLFMDTTSVELFVNEGEQVITNRFFPNEKSQQFNIFTSDGHLKIRNF
ESWKLNSSWK
>lcl|NZ_CP026520.1_prot_WP_042228754.1_2072 [locus_tag=PC41400_RS10650] [protein=PfkB family carbohydrate kinase] [protein_id=WP_042228754.1] [location=2328612..2329574] [gbkey=CDS]
MYDVVSLGELLIDFTPVGVSDNGNTLFETNPGGAPANVLAVLSRYGHKTAFIGKVGDDQLGHFLGKVLENNHIDARGLSY
ANNVNTTLAFVHLDKAGDRSFHFYRSPGADMMLSEDEVDLTLLNNAKVLHFGSLSMTHEMVKKATLKAVAAANEKGILVS
YDPNLRPLLWDSLERAKETIIEGMQYADIVKISEEELLFITGLTDLEKGSRELVERFHLKLLLITLGDKGCFYRLKHETG
YVKAKSVNAVDTTGAGDIFLGSLLHKILEKKITLELLTPGIIEEMISFANIAAGLSTTQKGAIPAIPELDDVNKIFEDNV
>lcl|NZ_CP026520.1_prot_WP_042228752.1_2073 [locus_tag=PC41400_RS10655] [protein=zinc-binding alcohol dehydrogenase family protein] [protein_id=WP_042228752.1] [location=2330211..2331167] [gbkey=CDS]
MKAAIVKEKGTIPVMGHFDFPVATDGQVVINVSATALSRVSKFRSMGMHYSSKVNFPIVAGLDGVGTLKDGTRVYFALPT
APYGSLAEQTIVNEKLTVRLPDGINDLTAAAIANPAMSSWAALVFRADFKPGQTVLINGATGASGSLAVQIAKGLGAKKI
IVTGRNESKLQVLEADEAIAFDMTADNGQNRFENALMSVVAEGVDVVLDYLWGDSALAIMSALARAHTDRTTRFVSIGTS
SGQENIHLPSSILRSSTIELVGSGDKSVSKADMLSAVKGVFDMAAEGKLKIAIKEFALEDIEVAWNAPLTPRPVVKVQ
>lcl|NZ_CP026520.1_prot_WP_042228750.1_2074 [locus_tag=PC41400_RS10660] [protein=MarR family winged helix-turn-helix transcriptional regulator] [protein_id=WP_042228750.1] [location=2331171..2331605] [gbkey=CDS]
MENEIFKALFDLVSIINRPDRDKKMIDNAGVNLEAVTFRVFVGIAHLQPTSVGELAAMMGKNYSSVSRQIDKLETAGLVH
TYPSSSDSRIRVSELTTYGEEVSTMIGLARERNMREALADWTPEERNGLLNNLRRLFESLKRFD
>lcl|NZ_CP026520.1_prot_WP_042228749.1_2075 [locus_tag=PC41400_RS10665] [protein=response regulator transcription factor] [protein_id=WP_042228749.1] [location=2331964..2332650] [gbkey=CDS]
MEKITILVVDDDSEIRDFIRIFLCNEGYHVLEAENGLQALEIVKTSSVQLIILDVMMAHMDGIRACLKIREISRIPIIML
SAKEEDIDIITGLTTGADDYMTKPFSLLELTARVKAQLRRQSYSVKKPEQDVIVVGDLIIDGAMHQVTVKGKEISLTPLE
FSILELMASHRGHVFSVDKIYERVWKESSSISDNTVVVHIRNLREKIEVNPREPRYVKTVWGVGYKIE
>lcl|NZ_CP026520.1_prot_WP_241688838.1_2076 [locus_tag=PC41400_RS10670] [protein=HAMP domain-containing sensor histidine kinase] [protein_id=WP_241688838.1] [location=2332643..2333761] [gbkey=CDS]
MNKLLSLIFGKRSIRVRMFWSFILSLTLAMNVSSLLNESLLNSLLAARHFPYLSIVLYGVLSTVSFIVLFILSFSLMTRR
IVQYLLSLAEGLDYIAGGNLRHRVPLLRKDELGRVAENINAMAEKLERQIAKERQIQKSQMDLITGVSHDLRTPLTNIIG
YLELLKNRVYHNEQEQERFVDITYNKAMQLKNLIDDLFEYTRLTTSDAKFVMGQFDIRELLSQIAAEFQPFAKEHGVTVE
TYLAFKSMPIPMDPGKIRRVIDNLLMNALKFSVKPGVVRVSLEARPPFVAIKIENEGTPISKEQEELLFGRFYKADHSRT
ANAIQTGSGLGLSIVKSIVERHNGTLSLVHSSGHFTFIIELPLTTAAHDDEN
>lcl|NZ_CP026520.1_prot_WP_042228747.1_2077 [locus_tag=PC41400_RS10675] [protein=carboxylic ester hydrolase] [protein_id=WP_042228747.1] [location=complement(2333758..2335203)] [gbkey=CDS]
MRFWEIGLIVVNFGLLYWAVIANKKGGRGLFIPLGTAYSLVIVQIVVEGGRWEMVPAYLTPLLLTAYFFIGKRREGGRSR
IAVTVQAVLLTVYLLVSVAPPAILPVFSFEKPTGPYAVGTTVYHWVDEQRRETYSSNPEDRRELMVQVWYPAAEGRGEKR
SPYLLNASKVTGGISSKTLGLPAFTFSHLGLIKTHSLLEAKLSDAESRYPVLIFSHGMNGYRNQNMYQVEELASHGYIVV
GIDYAYEAAASVFPDGRVAPSRTDSKLTSNAQYEQHIPLWTDDATFVLNQVEKLNRNDPAGRFTGKIDIDRIGMLGHSFG
GAVTLQMTKNDSRVKAALCMDGGIYGPPVSDKGLGKPFFQMYSEEIYNLVMASYDDVAKQLGGVSREVFEAPRKEYIQKS
RNALAGGGLSILIPGSKHASYSDLALFSPLPGLGGADPRRDHRIVNEFSVAFFDRYLKGKDDSALKDLEAKYPEVNYKVN
D
>lcl|NZ_CP026520.1_prot_WP_042228745.1_2078 [locus_tag=PC41400_RS10680] [protein=hypothetical protein] [protein_id=WP_042228745.1] [location=2336170..2336688] [gbkey=CDS]
MNQDSKIEVRKSILYNSLNELLRKKMPAICRDIEKAISDRNYTDVMNNIVCLLETHIWGQDTQWRVEAEVECYPCHESGR
PFAIGGFVFLEEVAGQKIIHFKTQMYNSFDESTIKKKDTIKINYDKVNDYEFIQPFIQETEEFLDQFYNVFMTGSDIFVR
EWRLAKSEQENL
>lcl|NZ_CP026520.1_prot_WP_042228742.1_2079 [locus_tag=PC41400_RS10685] [protein=DUF4386 domain-containing protein] [protein_id=WP_042228742.1] [location=complement(2337631..2338329)] [gbkey=CDS]
MLRKSTNSGRRSAVITGVLLISGLVTGIFSVVPVIDGADYLVKASTNGNQVLIGAFFQLLMVVAYVGIPILMYPILSKPH
KGLALGSVAFGIIAGVFIIIGVIILLLLLTLSHEFAKFGTLNGSYFQTLGGLLREGRDLVNHVATTLTSVLAMFLFNCIF
YQTKFVPRWLSVSGLIGSILSILASLLFMIRFIGLDATYMTLNVPIASQQLILAIWLILKGFNRVDVLEKKV
>lcl|NZ_CP026520.1_prot_WP_042228740.1_2080 [locus_tag=PC41400_RS10695] [protein=ABC transporter ATP-binding protein] [protein_id=WP_042228740.1] [location=complement(2338958..2340859)] [gbkey=CDS]
MPDPKSPDANAPREPASTGSAGTERPGQTADTAKSSGATEPKEGPRPGRLAAFKATLTYAKPHKLTFIGIFFCSLIAIMA
ELLQPYLVKIAIDDHLFGGSGAMRPLLIIGLFYLGLSVVNFAFTYIQNNLLQYAGQSIVASIRKNLFAHISKLSMSFFDR
YPIGSLVTLESSDTETISQFFTQVFLSLVRDGMTLVLIIFFMFQLDATLAWYSMLLLPVIALIAVLFRSMLRKAYQETRA
QLSRLIAYLAENLAGMSLIQTYNQQDEQFGRFSDRNGQYLKANLKEIRTNVLFNRSFDTFGNISVALLVWLGGLAFLGDS
IEFGVLYAFITYIRQFFQPINQITQQWNTLQSTSVSMERLWNIFSTEPDVKEPAGEGKAEVDWKKTAGKIDYNHVDFSYV
EGSKVLRNLDLHIKPGEMIGIVGTTGAGKSSLISLLTRFYDVNAGSVTIDGTDIRRIPQDTLHRMIGLVQQEPYLYSGTI
LDNVRLFRQEISREEVTAACKAVGAHPLIARLKDGYDTKLSERGSGLSAGERQLLSFARILVFKPRILILDEATANLDSH
TEQLVQKALSVVSEGRTSLLIAHRLSTIMHADRIIVMKHGEIVEEGTHGQLLAKKGYYEELYRHSRGMQAIGS
>lcl|NZ_CP026520.1_prot_WP_042228738.1_2081 [locus_tag=PC41400_RS10700] [protein=ABC transporter ATP-binding protein] [protein_id=WP_042228738.1] [location=complement(2340852..2342597)] [gbkey=CDS]
MGSQRLLRNYVKANLGLYSVGVAFIILSNVVQSFFPRVLGEFTDDLQLKVATSHTVAGYSLQLLGIGVAYGVLFGIGQYT
NARLGRRFEYATRQKLFRHFSTLSEHYFSRHGVGKLLSYVMNDVTAVRESISNGVNQTTNAIFFILSAFVMMSFSSIPFG
LIALSLAPMALIPFIVVFLGPRIRSRSLEVQESLATMTESAEEQFGGIRVTKTFAAEEIAYERFGRNVDDIREKQLRLVR
MSSLFQSLLPLMGSFSMIIAILAGGYMTLHGQITLGSFVSLTFYLRMMMTPLQQIGNVINMMQRSRASLERLNHLLAVKP
DIREEAKAAHLENGAGELEIRGLSFCYPGSDRRALEAIDLKVEPGRTLGIIGRTGSGKTTLVKLLLRIYDPPAGTIYIGG
HDIRRVTLESLRSQIAYVPQDGFLFSTTIRDNIAFSRRDAADDEMQQAAREAQIYSSITELPRGFETKLGERGITLSGGQ
RQRASLARGFMKRAPILILDDSVSAVDAVTETNIVRQIRAAHRDKTTLIIAHRISAVKHADEIIVLDEGRIVQRGTHKEL
LALPGLYASLYQIQEEGTRHA
>lcl|NZ_CP026520.1_prot_WP_042228736.1_2082 [locus_tag=PC41400_RS10705] [protein=S8 family peptidase] [protein_id=WP_042228736.1] [location=complement(2343010..2344227)] [gbkey=CDS]
MTTLISVLAFIAAAAVVYFIFRKKELPKEPHRPCQLIVKFKPETPEELIANLHKKYKCKVVEQCGTLGHQVLFTKRPLLK
TIKHYEKHEVLEYAELNYMYQAFEPAPSGGKKPGKGQSSSDVHALAIPNDPYFGLYQYGPQKISAPSAWDVAQSASSVVI
AVLDTGVELTHPDLVAHLVPGYDYVDNDPNPSDGNGHGTHVAGIAAAVTNNGIGIAGVAPLASIQPIRVLDNSGSGTVSN
ISNAIIYASFFGAKVINLSLGSPYSSTTLQYALDYAWGKGSVIVASAGNDGSTTPNYPAAYSTVISVASTDWADNISSFS
NRGSWVQVAAPGTAILSTFPGSSYAYLSGTSMSAPHVSGVAALLAGQGRTNAQIRDAILFNADPIYGTGTYWVYGRVNAD
RAVRS
>lcl|NZ_CP026520.1_prot_WP_241688839.1_2083 [locus_tag=PC41400_RS30260] [protein=hypothetical protein] [protein_id=WP_241688839.1] [location=2344446..2344940] [gbkey=CDS]
MSTNIRKMLIMNFVSSIIFIYIGIFVNLYIWQANTSIFEISWFNMVMFLGWGFSFVLGAKLLYHRTIRLLLALSAFGGTA
AFVMLTFLTLENRILWIALIGFPVGLMWGFYTVAQNLSVSFSGKGTEIASFFAANNLIASILSMVIPIISASLSAGSGMK
ARLS
>lcl|NZ_CP026520.1_prot_WP_241688840.1_2084 [locus_tag=PC41400_RS30265] [protein=hypothetical protein] [protein_id=WP_241688840.1] [location=2344940..2345692] [gbkey=CDS]
MFVFLIVMLIFSATMPRISLPRTETPRLGFWQQLQWKRVFGDRHSVWLILSILAGGVFLQFQNLFTLVFTFTVTQDKLLI
ALLNLLYTGAAFLGLILYRKFKWNDHFCLWLGVLLLSAGFVIAMFPLKPLLILSNVLTAFGMFFFAMIWNAQQFQCIEDY
APGRKAAFLIWRESMLVGMRIVLLVFIMSLDKLEGWLYISIIGVTLASLILIPVVQAKAVRLMAHGRKKREGTEPLGEEG
NLPRPPTFSA
>lcl|NZ_CP026520.1_prot_WP_009677256.1_2085 [locus_tag=PC41400_RS10715] [protein=HPr family phosphocarrier protein] [protein_id=WP_009677256.1] [location=2345895..2346152] [gbkey=CDS]
MEKEFSVQNPLGIHSRPAGALMKKAKAFTCDITIAKGDKSVNAKSIVGLLSLEMRYGDKVTVKASGEQEEEAVNELGTML
ESVLE
>lcl|NZ_CP026520.1_prot_WP_042228734.1_2086 [gene=ptsP] [locus_tag=PC41400_RS10720] [protein=phosphoenolpyruvate--protein phosphotransferase] [protein_id=WP_042228734.1] [location=2346161..2347915] [gbkey=CDS]
MIEERVTESVKGIAAAPGYAMGKAHLWQPQAEKAQRRTLPEEEIAAEVARLEEKTREALDELQCLKEETALKLGEAHAEI
FETHMLLLEDDDFVGAAAEAVRSERLNAEAAVEDAAKVLVELFENMSDAYMRERASDIRDVAGRLQRLLRGESAGDIGGG
GEAVVLFAHDLTPSDTARLDRSRIAGFVTAVGGRTSHSAIMARSLEIPAVVGMGAGAERVQPGDYVIVDGGEGTVHINPD
AGLLELYRSKQLRYEERRSRLGVYIGRPTETADGHHVELAANIGSPQDAQAARDGGAEGIGLYRTEFLYMGRDTMPTEEE
QFLAYKIVAEMFAGQPVVIRTLDIGGDKELPYLTLPKEANPFLGYRAIRLCLDRKDLFRTQLRAILRASAYGNVKIMYPM
ISNIEELRQANAVLAEARAELDAEGVKYAQDVEVGIMIEVPAAGMAADQLAREADFFSIGTNDLVQYMMAADRMNETIAH
LSEPFNPGVLRLIRHVITAAHAKGRWVGMCGEMAGSVVAIPLLLGMGLDEFSMGSGSVATARALMHKLNRSELSRLADEA
LELDTAEEIRRHIINRVPAVAEWL
>lcl|NZ_CP026520.1_prot_WP_042228733.1_2087 [locus_tag=PC41400_RS10725] [protein=PRD domain-containing protein] [protein_id=WP_042228733.1] [location=2348111..2350033] [gbkey=CDS]
MTLTTRSRALLKKIVYSHSPLRIKECAEVFQVSERTVKYDLESIRLWLQQEQIELLSKPSRGIWIDCSDTDRIHLLEKLD
AVGEHVFLNQQGRMRHMLIDLLLLDGYLTIGELVAKHDVSRNTVQADLALADQLLAGSGLTLNRTRHGIRVSGSEMRRRG
VLEHVIQDLLDGNDMFQIVQGVVQERRPHLHFSKLLEWFLQPVRDLDRVFEDVGSLVREIREKTQLLLTDNAIIGVFIRL
CIVIQRQRPAEEQPAADSGERTVASDQRSLAGYHLSISDLFRQKLTGLFAELELPLSEENIRYVSLPAIGLIPPLLKGRD
EEGAQLPDAFAVTKKLIDIVSESVHVRFRKDPDLMQHLHAHIADKLNKYSYGVAEPNPLLQEILRSYRTMFETVRSACGE
VFSPYGILLSDSDIAYIVLHFQAAFERRQEVQQYRALVVCGTGRGTSKLLKTVIENEIKTLQVAAFCSVMEFDKQVVGDF
DLVISIFPIEAHVPVVVVNPIPDKNDFKSIRLRLENLEKHGGAEGVSAGRSSSLQRPELSYLEQKFQDIIWKGFELSRAI
RTRFAPYLNEERLEGLTLHLLFMMNRIAFDTAYIQTDSALSAEGAEGELAEELKMLLSEHRIHATEGEVMAILRYFERSA
>lcl|NZ_CP026520.1_prot_WP_042228732.1_2088 [locus_tag=PC41400_RS10730] [protein=amidohydrolase/deacetylase family metallohydrolase] [protein_id=WP_042228732.1] [location=2350036..2351175] [gbkey=CDS]
MKADLLIHGGTVIDPSTGISGAFDVAVSGGKITGVFKPGTAVVEAGQTLDASGSYVSPGFIDIHTHIYEGVPAMGVAPDR
VGIDQGVTTVVDAGSSGAANFDDFVRRSIEPSKSRVYAWLNISAPGLVEGRSELADLTKLDPKEVLAVIERNRDAIIGIK
ARMSGSVVKTSGIRPLQIAKDTAREAGLPVMVHIGNAPPQLGDVLDLLEAGDVVTHAFHGKAGGILSPDGGLIPQAVRAL
ERGVLFDVGHGSSSFSFDTMKKAKELGIDPYSISTDLYLENLERGPVYSLALTMTKLLAMGYSLEQVVAWSTVAPAKAIQ
RENEIGTLREGAEADITVFRLVQEPVSLTDSEGTTVRYDQQLLPQYTLRAGAIHHVQHK
>lcl|NZ_CP026520.1_prot_WP_042228731.1_2089 [locus_tag=PC41400_RS10735] [protein=PRD domain-containing protein] [protein_id=WP_042228731.1] [location=2351159..2351524] [gbkey=CDS]
MYNINEVLEEIPQKTEASPEKLEAVKDLLTKLRPMSQEIGLSIPHTKAVIIGIHLLAFLRRVDQDEYLPEIEPEMFEEIS
AESIDLSKRLFAGYNMPPQRKLDDAEVFFLAVHFEAAKTNL
>lcl|NZ_CP026520.1_prot_WP_042228730.1_2090 [locus_tag=PC41400_RS10740] [protein=glycine-rich SFCGS family protein] [protein_id=WP_042228730.1] [location=2351538..2351903] [gbkey=CDS]
MENKVKVVIGDRLGKGQNVAKGVEAAGGVAIVIPGVGADMKLGDVMNKENADFGISFCGSGGAGAVTANTKYKYPMEFGM
RSIDAGVTAIREGKKVLGFGFMDVEELGRRLTEEYIKKNGR
>lcl|NZ_CP026520.1_prot_WP_042228729.1_2091 [locus_tag=PC41400_RS10745] [protein=DUF4312 family protein] [protein_id=WP_042228729.1] [location=2351907..2352245] [gbkey=CDS]
MFQEKPLTLVVSGMSETKQGAFQQALTRMKSQISKETSDVLLQIEPQDVEVVSAHLTQYREKFLGFLFPRTRTRYEVTLR
ITVKLKYVNLSEVEFTENKENLSKYQRLLHMR
>lcl|NZ_CP026520.1_prot_WP_042228726.1_2092 [locus_tag=PC41400_RS10750] [protein=DUF4311 domain-containing protein] [protein_id=WP_042228726.1] [location=2352323..2353108] [gbkey=CDS]
MNTLMIVFESIIIGLLVGFGVGAGAARMFHAPNVQGMGAFRTFGELNACEGDAVAHFSFGLGFLFNAWASVVGAGALTQD
VDHRIIPHWSAAALLSRNKKLEETLHDPRKMAWTGAIIGAVVVALLNSTAAAIPASLQSVATKVLVPAANWLINPIMPIV
FWTAAMDAGKRTGVWATMLGGFSHLVMGNAVPGIVLGILIGKGVDDSGWNRITKTMLVAVILLFIFSGFFRAFDVALLKS
MSLTTPQWLIDLHAFFGSVVK
>lcl|NZ_CP026520.1_prot_WP_042228724.1_2093 [locus_tag=PC41400_RS10755] [protein=DUF4310 family protein] [protein_id=WP_042228724.1] [location=2353110..2353760] [gbkey=CDS]
MENNLQRGFWYSEWAFVLFVACLSSGIFAGTHLYYVYHVGAFNDIAIVAMLEAGLKGGSYGAAAAFGASFLFARILEAPL
VGILDIGGSLQTGIGIGVPALMLGAGMTAPLTNFPLALLTGAVLGVAVGAVIILIRKFTINASNSTFGADVMMGAGNASG
RYLGPLIVISAVMASIPVGIGATIGAAIFYYYKKPIAGGAIIGAMILGAIFPIVTK
>lcl|NZ_CP026520.1_prot_WP_042228722.1_2094 [locus_tag=PC41400_RS10760] [protein=DgaE family pyridoxal phosphate-dependent ammonia lyase] [protein_id=WP_042228722.1] [location=2353837..2354937] [gbkey=CDS]
MSIFERYGCKQVINASGKMTALGASAVHKDVAAAVAEAAMDYVDISELMVQAGKVIAEVTGAEDGCPTTGAAGGIAISVA
AVIAGTSLSLIERIPLTDGLRNEIIIQKGHSVHFGASIPQMIAIGGGKVVEVGQANHVEKDHIAEAVTDKTAALLYVKSH
HAVQKGMQSIETMAAIAKDKGIPLIVDAAAEEDFRKYVAMGVEIVVYSGGKALEGPTSGFICGRADLMVACRAQYKGVGR
PMKIGKEGIIGLLTALRRYDNRPDESQQQRERMEWLIAELKDVPGVKGAITQDEAGRAIYRAQLTIDAKAAGKTAYELIK
ELESGSPAVYTRNHYSNLGIISVDPRPLLEGQERVIAQRIKEILQA
>lcl|NZ_CP026520.1_prot_WP_042228720.1_2095 [locus_tag=PC41400_RS10765] [protein=KDGP aldolase] [protein_id=WP_042228720.1] [location=2354978..2355697] [gbkey=CDS]
MAQPTIKLNVLAKTAENAKQIVEAAEGRVYVGLMVKNFPSVDEAVKTVEEFQAAGVPVSVGLGAGDPAMWSKVVDVSVRT
KPAHVNQVFPAAGYTLGAMRAVGSEDTLVNALITPSGTAGQVRIATGPTSVSFENDTISCEAAAAMLAEIGIHSVKFYPI
GGTDHLDELAAMVKASVAQGIKIFEPTGGIDIDTIGKVVEVCAENGAEIIIPHVYTSIIDSATGLTRIEDVKELMRAIS
>lcl|NZ_CP026520.1_prot_WP_042228719.1_2096 [locus_tag=PC41400_RS10770] [protein=HD domain-containing protein] [protein_id=WP_042228719.1] [location=2356019..2356678] [gbkey=CDS]
MEPQTVLEKAYQLAGEKLKDEPTGHDWWHIHRVTRLAKLIAKKENADEFVCEMAALLHDLVDEKLVDDKKRETEKLAEWL
ADNEVPGETIRHILDIIGTLSFQGGTGTKMSTLEGRIVQDADRIDAIGAIGIARTFVYAGARGSLIYDPQLEPRSEMTAE
QYREGKSTAINHFYEKLLKLKELMNTETGRQLAEERHEFMEQYLVRFYSEWNLGNGSGL
>lcl|NZ_CP026520.1_prot_WP_042228717.1_2097 [locus_tag=PC41400_RS10775] [protein=CsbD family protein] [protein_id=WP_042228717.1] [location=2356977..2357210] [gbkey=CDS]
MDSNVFKGKWKQLKGEAQKQWGKLTDDDMDVINGEKDKLVGKLQERYGHTKDAAEDEYHRWSSSYRDEAPADADRRI
>lcl|NZ_CP026520.1_prot_WP_042228715.1_2098 [locus_tag=PC41400_RS10780] [protein=phosphatidylinositol-specific phospholipase C/glycerophosphodiester phosphodiesterase family protein] [protein_id=WP_042228715.1] [location=complement(2357363..2358367)] [gbkey=CDS]
MKKHLTGVLALLLTLSLLPFQHASAKEAEPWTDQTLIAHAMGGISGETLTNSYEAFLINYEKGHRLFEVDLNLTSDGQLA
ARHDWLRYLSDRLQPTLPDNLKDGPVTLEEFKRHKILNKYTPLDLDLLLQLLEQYPDAYIVTDTKSEDPQVVQKQFQLIV
QKAKAVNPALLERIVPELYEIDMINQVKQIYDFPNYFFSIYMTSYSEKEIISLVKKNNIKAVAMPVERATSTMIGGLSNL
GVPVYVHTVNELKEIKKLKKIGVHGFYTDFLTYREVAPDSAAAKTQTAVKAASAAPAGDQASSVVQAGDKPAAGAPKETL
MQKLASLVIQVFEK
>lcl|NZ_CP026520.1_prot_WP_042228713.1_2099 [locus_tag=PC41400_RS10785] [protein=chromate transporter] [protein_id=WP_042228713.1] [location=2358749..2359423] [gbkey=CDS]
MNLLKFSAARLKLLLQLFGSFARIGPVSFGGGFAMLPILEREIVLRRRWLDEDEFGEVISVSGVAPGGVGVNAAAFIGYK
LGSVPGMLAAVLGIMLPTFVLVLILAIGFSAFRQYPKVIAAMEGIQMAVLALIAYAGLKLLRSSVFDGASLTVFAGSIAA
LLLTGIHPLLVMVLGVTAGCVLVLAKGHFGFAVRFEPKKRRRERSADPAVPRYPELECYFGEGI
>lcl|NZ_CP026520.1_prot_WP_042228710.1_2100 [locus_tag=PC41400_RS10790] [protein=chromate transporter] [protein_id=WP_042228710.1] [location=2359438..2359974] [gbkey=CDS]
MLITLFWTFLKIGILSFGGGFAMIPAIQHEVELQGWLTPEQFSEAVAISGMAPGPLATNCAIYIGKMTMGLPGAAAAVLG
MVLPSIVLIAAVARFFYAIRTHRTVKTVFYGLRPVVAALVVVAALRFGFTYFRAEWWTWKTGVSAAIFLLALLGLTRYRM
HPLSVILLSALVGVAIYG
>lcl|NZ_CP026520.1_prot_WP_042228708.1_2101 [locus_tag=PC41400_RS10795] [protein=ROK family transcriptional regulator] [protein_id=WP_042228708.1] [location=2359967..2361178] [gbkey=CDS]
MVNSPISERIGNRKQKELFELIRQGGTVSKIELLEESGLTVSTLTRLLDELCAQGLIAESGHGPSTGGRRPVLYRIQADY
GCAFGVDISRTLSRLVLTDMQMKVLESRTWEMKEGLTPDVLLASIADEAEAMLQHRGFSPGHVLGMGVGAVGPVNREEGI
ILNPRYFPTPGWHGVPVRKMLEERLGLPVLLDNGANTALLAESWSRQKRSFRHMLYVHLGVGLRSSMLSEGKLIYGAVDM
EGSIGQMIIQTDGVAPRLQEGNYGALESYASIYAVEREMRARLKQGRSSVLSALGSPDALTFREIIQALHSRDPLAVEVI
TQAAVYFGIGLANLLNMLHPEKVVLGGPLISTNDLFFYTATQTAIKKTYYYPDYQVLFSKGSFGEEALAIGAAGLVIRQL
SAI
>lcl|NZ_CP026520.1_prot_WP_042228706.1_2102 [locus_tag=PC41400_RS10800] [protein=SMP-30/gluconolactonase/LRE family protein] [protein_id=WP_042228706.1] [location=2361281..2362123] [gbkey=CDS]
MEKLFTSSVFTPPHGFTGGIEGPACDAQGNLYAVNYAKEGTIGKVTPAGQSSVFLELPNGSIGNGIRFTRAGEMLIADYK
NHNVLKVDMNTKAVTVWAHGPGMNQPNDIAISANDIVFASDPNWSNNTGTLWRVTPGNVTLLESGMGTTNGIEVSPDGKR
LYVNESVQRRIWVYDLTEAGAISNKRLLIEFPDFGLDGMRCDSQGNLYSTRFGKGTVVKISPEGNILQEVVLHGKNCTNL
TFGGPDGRTVYVTVADNGNVEKFRTDAPGRCRVLSGVGPS
>lcl|NZ_CP026520.1_prot_WP_042228704.1_2103 [locus_tag=PC41400_RS10805] [protein=sulfatase-like hydrolase/transferase] [protein_id=WP_042228704.1] [location=complement(2362208..2364307)] [gbkey=CDS]
MQKGIHFIARRLHLFVWPLLLVVFIEFLNRGGVMAALSWVKHSPGEMVLNYLLTFCLFLVFTALTGRSRIAFWIVAAFML
ICGLISGIKFKILGVPLLPWDLFLSSETTDIVQYLNGLFDKTIVIGIIVFLAASVALLYFVPWTHSRFRWKENIVLGVLG
IALACVVYFDKPVPIKSALHISTIPWNQADNYHVNGFMLSTVMNIEQIIIDAPQGYSQANMDNFVKRVDRRTNIDPNVKP
NIIVILGEAFWDPTRMKNVTFSEDPVPFIHHLQQNYSSGYMLSPQFGGGTANVEFEVLSGNSIRFLPEGAIAYNQYMNRR
VDSLASILARQGYQTAAINPFHNWFFNSRNVYQNFGFSKFISQEYFEPEYNGPYLTDKAVMNKIIEQVDSSPGPAFVFAN
TMENHAPFTPDKFKQNTIKASGNFPEETKGMIETLAQGVHNTDAALKMLVEHYEKKGEPTIIMLFGDHLPAMGNNFQAYK
DTGYLKDDDDSAAVLDKMYTLPVVVWDNYLPKHQDKLNMSASFLGPYLLNLAQKEGTPYTDYLFSLSKKFPMIPPRNHYD
RLKLTEQDVTEYRMLQYDVLFGEQYSFNGFRDKIINPKYTLGYGEMSIDSVSPLHQSTAGGSGGDSSVEISGKNFAPSCV
VYLNDKALKTTYEGRQSLSAVVPKDAIEPGKPVNLQVKVIDSKNLPIQTTPIKQVNELK
>lcl|NZ_CP026520.1_prot_WP_009677255.1_2104 [locus_tag=PC41400_RS10810] [protein=cytochrome c oxidase subunit 2A] [protein_id=WP_009677255.1] [location=2364518..2364691] [gbkey=CDS]
MAKLAPAKEKQDIKPAPQSGANEKEPELKGAFASVMILGAFILITWFGVFALYLIRS
>lcl|NZ_CP026520.1_prot_WP_009677258.1_2105 [locus_tag=PC41400_RS10815] [protein=cytochrome c oxidase subunit II] [protein_id=WP_009677258.1] [location=2364746..2365216] [gbkey=CDS]
MHIHRLEKIWIVFGISMLIVFLSVIGVSAFAMGMNPPTGHAHTIDPAKVDLTPPFDQPGLKKVGENEYEAVMTAFAFGYK
PDKMEIPAGATVHFMITSKDVVHGFEIPGTNVNMMVLPGEVNHLTYKFDKPGSYLVLCNEYCGGAHEWMKTTIVVN
>lcl|NZ_CP026520.1_prot_WP_042228700.1_2106 [locus_tag=PC41400_RS10820] [protein=b(o/a)3-type cytochrome-c oxidase subunit 1] [protein_id=WP_042228700.1] [location=2365246..2366919] [gbkey=CDS]
MNGTETQPARQAAAAFNPKDSKLVLAHILFAFGALLLGGIAGVLQGLVRGGMIKLPAGIGYYQLLTAHGVLMALIFTTFF
IIGFLYSGISKTLGGELLPLPRRLAWLGFGLMAGGTALGTVFILLNKASVLYTFYAPLKASPYFYIALTMVVVGSWMSGF
GIFINYRYWKRTHKGQLSPLFSFMAVITMLLWQVATIGVAAEVLLQLIPWSFGWVDTVNVLLSRTLFWYFGHPLVYFWLL
PAYICWYTIIPKVIGGKIFSDALARLSFILFLLFSIPVGFHHQLTEPGISSFWKFLQVILTFMVIVPSLMTAFSLFATFE
TTGREKGAKGLFGWFKFLPWKDVRFFAPFMGMLIFIPAGAGGIINASNQMNQVVHNTLWVTGHFHLTVATSVALTFFGIT
YWLIPAITGRQLTQRLHKLGIWQTILWCVGMFFMSGSMHTVGLLGSPRRTAYTTYENHPDTVLWMPYHVAMAIGGMILFV
AVVMMIYNVVALMRSPKGETEYPIGEVRDASEPTPKLFERWGLWLGVTAVLILVAYTIPVAQMMGGDTPGSIPFVTW
>lcl|NZ_CP026520.1_prot_WP_042228698.1_2107 [locus_tag=PC41400_RS10825] [protein=FAD-dependent oxidoreductase] [protein_id=WP_042228698.1] [location=2367063..2368307] [gbkey=CDS]
MDLITGNPLWPHTSNEKKQFPRLEEDIECEVLVIGSGMSGAMSAYMLTKAGFDTVLVDSREIAGGSTMANTGLIQFFNDK
TLTSCIHTYGEENGTAFYKLCEEAVRTLGRICEEEGIDAQFQRRESLYMASTEADVPMLEEECRTLNKYGFEVEYLRPSD
IADKFSFTKPGAILAYGDAAINPYACVNGLIEVSAARGMRVYARTEIDSRLETANGTVFYTKERRRILAKYAVYSTGYET
QAMKRNRNAKLLTSYAIATQPVGEFPGWYKECMIWETARPYLYMRTTVDGRIIVGGEDEDMLPEEERINRLAVKTDILLD
KAKALFPEFDLRAEFSWAAVFGETHDGYPLIGVQEGFPNSMFSLVYGGNGTVYSVLGAEMIVNRLQGLPHPGEWLFRFDR
PRHTPPASEVKQVG
>lcl|NZ_CP026520.1_prot_WP_042228695.1_2108 [locus_tag=PC41400_RS10830] [protein=Gfo/Idh/MocA family oxidoreductase] [protein_id=WP_042228695.1] [location=2368422..2369423] [gbkey=CDS]
MKVAVVGCGTMGNVHGTDYHRMPGVELTGVCDLDRETARDLAGHLQTKAYFSFEDMMAEAEIDAVSIALPTNLHKEYVMK
AADLGKHVICEKPIALTLEDAEEMIAYCQKKGVRLFIGHVVRFFPEYADLKRKLEAGVTGEPGVAHLKRVGSHPGTVKDW
YNDFDKSHGVIMDMMIHDIDFARWAYGEVQEVFALNRRSEEQDYALVTLRFAGGAIANLEGHWGYPGAFATAAEFAGTTG
IIRLNSRETDSLKLLRSSAAPAGARGVQVPKSPSYQDPYYYELEHFLACIREDKEPLVSAEDALRALEIVLAALESVRTG
QPVRLPLKGGVTG
>lcl|NZ_CP026520.1_prot_WP_042228694.1_2109 [locus_tag=PC41400_RS10835] [protein=Gfo/Idh/MocA family oxidoreductase] [protein_id=WP_042228694.1] [location=2369420..2370406] [gbkey=CDS]
MRKLKIGMISFAHGHAFSYFNALNALPEVEVAGIADEDKNRVAKITDTYGIPYYSEADELLRTDIDAVVICSENSRHAEL
TIASAKAGKHILCEKPLGISTAEMEQMITVCRDHGVQLMTAFPCRYIPAVMQAKDAIDRGEIGEIIAIKGTNRGSMPGGW
FVDRKLSGGGAVFDHTVHVMDLMRWFLGSDVREVFAYADTRFHELDIDDAGMVHVRFENGVFAVLDPSWSRSKSFPTWGD
VTMEITGTKGVISIDSFAQKNEVFSDVTGKAQWSYWGDNMDEYMIKGFVDALLGGKPVPVTGEDGLKAAAVGLAAYESAQ
AGKPVVLD
>lcl|NZ_CP026520.1_prot_WP_053228684.1_2110 [locus_tag=PC41400_RS10840] [protein=DUF1516 family protein] [protein_id=WP_053228684.1] [location=2370655..2371008] [gbkey=CDS]
MSDKLLNIFHQSHSGSWAFIVILFVVSAIFYRQKITPMILRLFYLIMLVSGIGMLIGYGFPVKYVVKGIIAIVMIGLMEM
ILGRMKRGQGKESSTKILFVVWAVLIVIVVLLGYGIF
>lcl|NZ_CP026520.1_prot_WP_042228693.1_2111 [locus_tag=PC41400_RS10845] [protein=KinB-signaling pathway activation protein] [protein_id=WP_042228693.1] [location=2371085..2371720] [gbkey=CDS]
MLKPMLFLFWSTMLGGLVLGIAVWGIGKAVFPDFVSMDVAVSGASEFSMIIMATLTMCMISVVGYFAYLMTNYYMIGILR
NRKGAWSGLQLFFIVMAIANLFFLRYQGFAEGKESLAMYAGIPVLIAAAALIVSQVKVKLTNASARIPTLFFMVVVTCLE
MLPAFSINDLTSLLYVSLVLLACNAWQILIFHKLTNTKKHQQMKKAQVKKA
>lcl|NZ_CP026520.1_prot_WP_042228692.1_2112 [locus_tag=PC41400_RS10850] [protein=tetracycline resistance MFS efflux pump] [protein_id=WP_042228692.1] [location=2371761..2372936] [gbkey=CDS]
MNKRVVFIMLLMFTVYLGFGLIIPVLPELLHGIPEGETHLGWVLSIYSIMSFLVSPLWGGLSDKIGRRPVLLTGIFGFMI
SFVLLALAGTNLPLLYLSRILGGICSGALSGVALAYVADITTHEERTKSMGFVGMSIGLGFIFGPAIGGTLSVVSLSLPF
WGAALLSAGIFCAGFLKLKESLPDNRGQNQGDKPSRWSLLRGKLGYLYVMTFFVTFSLAGLEATFQYFQITRFGVTPLQI
GLMFMFSGFADALVQGGLVRVIAKKKKEKLAILCGLFASAVGFLLVLGSDSFWTATLALTVFSAGNAMIRPCINSLLTQR
TRHGQGVTTGLSSSMDSLGRIAGPITGTSLFGIQHGLPFAAGAVVSVLASLLLLRFSQLDRESKDELPETV
>lcl|NZ_CP026520.1_prot_WP_042228689.1_2113 [locus_tag=PC41400_RS10855] [protein=DNA topoisomerase III] [protein_id=WP_042228689.1] [location=2373151..2375298] [gbkey=CDS]
MSKSLVLAEKPSVAKEIARVLGCGQKHKSYYEGPGHVVTWALGHLVGLAEPEDYDTRYQKWNLEDLPIIPGRMKLKVLKE
TSHQFRAIADLAKRKDLAELIIATDSAREGELVARWIMEMIRWNKPFQRLWISSQTDKAIREGFANLKPGKQYDRLFQSA
VCRAEADWLIGLNITRALTSKYDAQLSAGRVQTPTLGMIMDREKEIRNFRSVEYWTVQADFGAFRGTWRQRSGESRLFER
ERVDALVQRVSGKSGKVVSVKKTEKTEPQPLAYDLTELQRDANRRFGFSAKHTLSTLQRLYEQYKLVTYPRTDSRYLTSD
MTGTLKERLQSVAVGPYAPLAKPLMKGQLPVSKRIVDDSKVSDHHAIIPTDEYVNLSALSADERKLYDLIVRRFLALFYP
AARYETTAMVVEVDGEPFHVAGRVVKDAGWRSVYGPGTAADSGDGDEEDGGEGADRADLQELPALSAGDSAGTVRCRVGT
HRTQPPKRHTEATLLTQMEKHHLGTPATRAEIIEKLVSSDTIEREGGSLHPTGKGTQLIELVAEELRSPELTAKWEQELE
DIARGRGRTETFLSGIRDMTKQLVTGVRTSQTEYKPHNLTNSRCPECGERLMERNSKRGKMLVCANRECEYRRSAEKQLV
NRRCPQCHKKMELKTGKAGKYVQCLPCNVVEVLGESTGKVAKHQQQKLVKQYSKQEDLGSSLGDALKAALAKKDD
>lcl|NZ_CP026520.1_prot_WP_042228687.1_2114 [locus_tag=PC41400_RS10860] [protein=Dps family protein] [protein_id=WP_042228687.1] [location=2375469..2375924] [gbkey=CDS]
MKTLTQVKTVTQLNHLTADWTVLSMKLRNFHWFVNGPHFFTLHAKFEEFYNEAAQHIDEIAERVLALKGKPVATLAEALK
LARLQEAAGSETAEQMVEAVRGDFERLVGFLRESIAAAEEEGDEPTADMLTGLQASLQKHVWMLGAFLGRA
>lcl|NZ_CP026520.1_prot_WP_042228685.1_2115 [locus_tag=PC41400_RS10865] [protein=alpha/beta fold hydrolase] [protein_id=WP_042228685.1] [location=2376084..2377769] [gbkey=CDS]
MKKHLPIWQSAVIASMLLAGAVPAAAASVDTPGAAGSVSASDTKAATVLAPLRLVSETAGAKVAWNNEERTVTLSLGSDT
ITLAIGKSTARINDKEVELGAAVQIKDEKAFVPLSFLTEALHTAVSWDPQTDKLIIPKGDLKAAASAFASHWLQGDNESA
VSYFNEDLKKSLNANQLHSISQYYTNMYGPITGQLGAGTASNSVHENVYLTYETQSTVPFQMTVRFDKQERIDDLYIPPY
YSPDTYRKPSYDDSSAYVEKEVKLGEGRLVLPGTLTLPNKPGPHPVVVLVHGSGANDRDESIGGTKLFRDMAVGLAAEGI
ATLRYEKVTREHPFKSATPTFTVQDETVADAVYAVNQLAKTEGIDPQRIFVAGHSQGGMIVPRIIDGVKDVKLAGAVVLA
GPTNSIEDVVLEQSKYQLEVSRERGLDTSILEQQVAFWEQQLKLIKDPQYSVDNLPKDLALSNAYWWMDFRNYSASEIAK
KQDKPMLILQGDNDAQVLAYHLDKWKEALKDRSNVEFKLYPKLNHVFAEVDEKSTGAEYNEPANTPASVIKDIADWVKKT
K
>lcl|NZ_CP026520.1_prot_WP_042228684.1_2116 [locus_tag=PC41400_RS10870] [protein=nitronate monooxygenase] [protein_id=WP_042228684.1] [location=complement(2378173..2379288)] [gbkey=CDS]
MNLSSRSNRFCTLFQIRYPLIQAGMAGGSTTPALVAAVSEAGGLGTLGAGYMTPEQISAAIREIRSRTDRPFAVNLFILP
EPGQGPHEQEQRQKADEAWRLLAPLREKTGTREERPPAVWSESYREQLAVLLEERVPVFSSTFDVPDAATLAECARLGIR
TIGTATTVREAAALEEAGVDAIVAQGSEAGGHRGTFPGDSDGEPPLIGTMALVPQIADLVRVPVIAAGGIMDGRGIAAAL
ALGAEAVQLGTAFLACAESGAHAVHKEAVLAARDESTVLTRSVSGKHARGIRTALLDELRPLDSRLPSYPAQNALTQGIR
RSAAKAGNPAYMAMWAGQAASLARRSGAAQIIEELVRETGEVLNRLSGQKL
>lcl|NZ_CP026520.1_prot_WP_042228683.1_2117 [locus_tag=PC41400_RS10875] [protein=NAD(P)H-quinone oxidoreductase] [protein_id=WP_042228683.1] [location=2379570..2380571] [gbkey=CDS]
MKAVQIKQYGGPEELYIGERPDPVAGEGELLIQVKAAGLNRADLLQREGKYPAVPGESDILGLEMAGVVIGVGSGVTGWR
EGERVCALLPGGGYSERVAVPAGMAMRLPANLSYAEGAAIPEAFLTAYLNLVKLAGMTEGDYVLIHAGASGVGSAAIQLA
REFGAHALVTAGSSEKLEFCRELGAEGCWNYREGSFVPWLNEATGGHGSDIVLDLVGAPYFTDHLAGAAVGGRIVLVGMT
GGSEVPMESFDLNPLLMKRISIIGSTLRSRSREDKIRLTQQFWQQASSGFADGALQPVVDRIFPWSRVREAHEWMEANRN
KGKVILDITADAE
>lcl|NZ_CP026520.1_prot_WP_206735009.1_2118 [locus_tag=PC41400_RS10880] [protein=fibronectin type III domain-containing protein] [protein_id=WP_206735009.1] [location=2381281..2383104] [gbkey=CDS]
MDLNWNSTGAKSYKVKRSVSPGGPYALLASDLTSTTYTDTTVTNGVTYYYVVTAVNEAGESVPSPEKSIKPEATKYTGGL
LDRLWLKTGPAVNNPTGTERVMTDNDATTRYLLTPGILVWHTFQSPKEINSVIIQGTGGVTVEFYDANDNLLSSYTPITL
DGVESLPTPVKNVTTALLKKNTNAFVNEWNLFESPSAKPLSTVINWIQAGDRTVDLYWDSKGAKTYNVRRSATKGGPYTL
VANTKELSFTDTTVTNGTTYYYVVTSVNEAGESAPSPEKEIKPDATKYTGGLLDRKILKTGTAISNPTGSERIMTDNNIA
TQFVLRGMAWYNFDTPKNIEAVIVNGSSGAIVEFYDTNSNLLFSYTPAQNDTVETLAAPVKNVSVVVLKHTNGFVKEWNV
FGKAVEPPVAAPLNLTAAGGDKKVTLAWNSVNNATSYNVKRSTTAGGPYVTVGSVTGATYTYTDTNVVNGTTYYYVVTAV
APAGESAPSNEASATPKAGDVVIPPEEESGDSAILVITLNNGTEKEYDLSMNEVNAFIAWYEGRAAGKGPVMFAIDKHEN
NKGPFKNRKDYILYDKIITFEVNAYDNGKGGNGGTTPPVNPEDPSNY
>lcl|NZ_CP026520.1_prot_WP_042228679.1_2119 [locus_tag=PC41400_RS10885] [protein=divergent polysaccharide deacetylase family protein] [protein_id=WP_042228679.1] [location=complement(2383296..2384108)] [gbkey=CDS]
MPGIRFGSCRLLVWTLLLAGLSIFTGLEGAASAHVDEPPAVRKAAIIIDDLGNNMKGTDAILDLPFPITAAVMPFLPSSH
SDAEKAFVKGHDVIVHLPMEPLSGRKSWLGPGVITTDLSDAEVRKRVNAAIDDIPHAIGVNNHMGSKATGDERVMRLVME
VLRERGLFFLDSHTNYRSIACKEAKKVGVPCLENHLFLDDIHTKSKVTQQIHLIGAHSKAHQPCIAIGHVGAKGEITASA
LKDTLPKLKDTLSFVKLSQLYKPPAPETAP
>lcl|NZ_CP026520.1_prot_WP_042228677.1_2120 [locus_tag=PC41400_RS10890] [protein=hemolysin family protein] [protein_id=WP_042228677.1] [location=2384477..2385811] [gbkey=CDS]
MDSDPLPVLLNFVLIVVLVLLNGFFVAAEFAMVKVRASRIDALVLEGNVKARFAGKIVQNMNAYLSACQLGITLTSLALG
WIGEPTVASVLEPLLVNIGVPPVIIHTISFIIAFTVITAFHIIIGEQFPKTYAIRKSEDVTLWVAGPVLIFYKIMYPFIW
MLNGTSNWLLRKSGIEPEGDHDTVHTEDEIRLLMKESHRNGLIDNTELTLVDNIFEFTETNAREIMIPRTEMVCLHAAHS
FEENKEMAIQELYTRYPVCDPDKDNIIGFVHIKDLLKASDQLDDIRKIIRPLMTVPESMPISTLLKLMQKRKTEVALLID
EYGGTSGLVTVEDILEEIVGELQDEFDHERPVMEKKDEFTYSIDGMMLIEEVNNHFGTEIETEHYDTIGGWIYSQVEIPP
MKNQRIHLGPLQFIIDEVEHMRIVRLIVRKEPPEEQVLLQQVVS
>lcl|NZ_CP026520.1_prot_WP_206735010.1_2121 [locus_tag=PC41400_RS30080] [protein=YjcZ family sporulation protein] [protein_id=WP_206735010.1] [location=complement(2385908..2386000)] [gbkey=CDS]
MSGYGAGIGFTSTAVILVLYILLVIVLATF
>lcl|NZ_CP026520.1_prot_WP_042228676.1_2122 [locus_tag=PC41400_RS10895] [protein=DsbA family oxidoreductase] [protein_id=WP_042228676.1] [location=complement(2386241..2386876)] [gbkey=CDS]
MIIDVFQDTVCPWCRIGKSYLTEALGLWKGEPVSVRFRAFRLDSSIPPKGVPFTEAMNRKLQDPSGEKLQAVLDGVTRAG
RQAGVPFRFDRILSMPDTRLSQQLIAAVPESSKSLVLDRIYRAYFEEGLNIGMESVLLRIAGECGLDPAETAKRLYHEDV
ENSVRADYALAKRLNINGVPFFILNKKVGFSGARPPEALLDIMKRVSENRE
>lcl|NZ_CP026520.1_prot_WP_042228675.1_2123 [locus_tag=PC41400_RS10900] [protein=MBL fold metallo-hydrolase] [protein_id=WP_042228675.1] [location=2387068..2387964] [gbkey=CDS]
MADKKDKDKTRRGESVIGQIDRTLVPDKAIALWHLGQEGMAFKAAGIIGYIDPYLSNYIDDEFGNSPRQFETPLSPELIG
HADLVLITHDHPDHLDPVSVRAIAIASPQATWICPAPVVSQLTGLGIAPDAVIGALAGQMIDFRDMRITPVAAKHEEFET
DENGNHKYLGYVIELNGVTLYHAGDTIGYAGLLETLRPHRIDIACLPINGRDIKRYEANLMGNMNFREAADLGVGIGADL
IVPMHYDLFAFNTENPAYFVDYVHRTYPYQKFKLFVPGERLIYYRDDEMKESGGGNGD
>lcl|NZ_CP026520.1_prot_WP_042228674.1_2124 [locus_tag=PC41400_RS10905] [protein=Fur family transcriptional regulator] [protein_id=WP_042228674.1] [location=2388207..2388620] [gbkey=CDS]
MKAEEIIQAMAEQGMRITDQRRTLAELFAKSEGYVTPKAVYEIMGKTYGGLSFDTVYRNLRIMHDMGVLEQFVFEDGIKF
RARCSEMDHHHHLICLQCEKTIPFVFCPMDEVKGLPDDFKAVKHKFEIYGYCKDCQK
>lcl|NZ_CP026520.1_prot_WP_042228672.1_2125 [gene=yidD] [locus_tag=PC41400_RS10910] [protein=membrane protein insertion efficiency factor YidD] [protein_id=WP_042228672.1] [location=2388617..2388856] [gbkey=CDS]
MKKVLQAPIHAYRKYVSPLKPPTCRFYPTCSQYALEAIERHGPLYGSWLAAKRICRCHPFHPGGVDPVPPPRKEGRNTG
>lcl|NZ_CP026520.1_prot_WP_042228671.1_2126 [gene=metG] [locus_tag=PC41400_RS10915] [protein=methionine--tRNA ligase] [protein_id=WP_042228671.1] [location=2389306..2391324] [gbkey=CDS]
MDNNRKTFYLTTPIYYPSDKLHIGHAYTTVAGDTMARYKRLRGYDVMYLTGTDEHGQKIERKAKEKGQTPQAFVDEIVDG
IKSLWSRLDISYDDFIRTTDQRHKDAVQKIFDRLTQQGDIYLDTYEGWYCTPCESFFLENKLVDGKCPDCGRPVEWVKEE
SYFFRMSKYADRLLQYYEENPEFIQPESRKNEMINNFIKPGLEDLAVSRTTFDWGIKVPGNPKHVIYVWIDALTNYITAL
GYGSEDPSKYERYWPADVHLVGKEIVRFHTIYWPIMLMALDLPLPKKVFAHGWLLMKDGKMSKSKGNVVDPVTLIEHFGL
DALRYYLLREVPFGSDGTFTPEAFVERINYDLANDLGNLLNRTLAMLGKYFDGVVPDFQPGVTAFDKELIATAKETVETV
EAQMEKMEFSTALSAIWALVRRANKYIDETQPWALAKDDVNHGTLGSVMYALFETQRIVSVLLQPFLTRTPAKIWEQLGI
QDGALTAWDSVHTFGLLPSGTKTAKGDPLFPRLDMAEEVAFIAAAMNGGQAPAGEAAAEPASAAAEPAPAVKTPAEDAKP
APEAKPEIGIDDFAKVELRVAQVTAAEPVKGADKLLQLQLDLGYEQRQVVSGIAKYYKPEELVGRKVICVTNLKPVKLRG
VQSQGMILAASEGDGLTLATVPDSMPNGAIVK
>lcl|NZ_CP026520.1_prot_WP_042228670.1_2127 [locus_tag=PC41400_RS10920] [protein=metal ABC transporter substrate-binding protein] [protein_id=WP_042228670.1] [location=2391539..2392510] [gbkey=CDS]
MKRRRILTGLTLAISLLLAGCGGGQKSAVAEDKINITTSFYPLYDFAVKVGGAHVNVINLVPAGVEPHDWTPKGRDIKNI
SDSQLFLYQGAGFEGWVDDFLKSLKDSKVKPVVASTGVELLKAGEETHDHDHGKDEHGKDEHSHTGEAGKDTDPHAWLSP
KNAKKMSENIKNALVEADPAHKSDYEANYSAYAAKLDALDKQYKDSLSGLPKKEIAVSHQAFAYLCHDYGLTQMAIMGLS
PDAEPTAQDMKKMREYIGEHGIKYIFFEELVSDKLAKTLAKDAKVETLVLNPLEGLTEEQVKQGKDYISVMEENLKNLTK
ALQ
>lcl|NZ_CP026520.1_prot_WP_042228669.1_2128 [locus_tag=PC41400_RS10925] [protein=metal ABC transporter ATP-binding protein] [protein_id=WP_042228669.1] [location=2392577..2393350] [gbkey=CDS]
MDTWKPDGCHNKIVEFSGISFSYGSKPVITDLNFDVLERDFVGLIGSNGAGKTTLLRMLVGLHKPTEGDIRLFGQPIRQF
RDWERIGYVPQKNSLNPLFPATVREVVMSGLYTKKKLYKRLTKEDMRRCDDAMEAMRIEDLAKRRIGELSGGQQQRVFLA
RAIANNPELLILDEPTVGIDAESQAVFFNMIRHMHQHHHMTFIMVSHDREMMRSYLGSEPQTQSGGLKFFVKHTHEPENC
EETDLTHSTVRELQTSQ
>lcl|NZ_CP026520.1_prot_WP_042228668.1_2129 [locus_tag=PC41400_RS10930] [protein=metal ABC transporter permease] [protein_id=WP_042228668.1] [location=2393402..2394205] [gbkey=CDS]
MDIFTEPFFVRALIGGILIGLTAPLMGVFLVLRRLSMIGDTLAHVSIAGVALGFLLGVYPIGMGLLFALLASFAIEKLRK
AYKTYAELSIAIIMSGGVALATLLFTLGKGYNINVMSYFFGSIYTLDSSDLWVVFGVCILVLAVVALHFKELFLMFFDEE
AAGVSGLPLKVYNIMITMLTALVISVSIKIVGALLVSALLTIPVACSLLAAKSFRHSVVLSVLISEIGVVAGLALAAWYD
FAPGATIVLSLIVVLMVQLVWKTVRRA
>lcl|NZ_CP026520.1_prot_WP_042228667.1_2130 [locus_tag=PC41400_RS10935] [protein=cytochrome c biogenesis protein CcdA] [protein_id=WP_042228667.1] [location=2394240..2394953] [gbkey=CDS]
MQHVNVWLAFWAGFASFISPCCLPLYPSYLSYITGISVSELKSGQTKGAARLQTMAHTLAFILGFSIIFYSLGLAASSVS
EFFVTNRDLLRQIAALLILVMGLFLLGIFQPKWMMRERKLQLKWKPAGYLGSVLIGIGFAAGWSPCIGPILSSMLVMAAA
EPDSWLPLISAYSLGFAIPFFVLSFFIGSTKWILRYSAMIMKIGGGVMIVFAILLYTNQMYKITIWLNNITPQWLQF
>lcl|NZ_CP026520.1_prot_WP_042228665.1_2131 [gene=splB] [locus_tag=PC41400_RS10940] [protein=spore photoproduct lyase] [protein_id=WP_042228665.1] [location=complement(2394977..2396035)] [gbkey=CDS]
MATEQMTRTRSTALFVPELVYFEPDALLYPKGKRILEWAQNKGLPIHMTTSHNRITNFPGETELEKYRIAKRTLVVGIRK
TLKFDTSKPSAEYAMPIATGCMGHCHYCYLQTTLGSKPYVRVYVNTDDILKAAKQYIEERAPEITRFEAACTSDPVGLEH
ITGSLKEYIEFMAKEPLGRLRFVTKYHHVEPLLDAKHNKHTRFRFSVNADYVIKNFEPATSAFHERIEAAGKVARAGYPL
GFIIAPIIWHDGWEKGYEELIVKLHDALPPEALDDLTFELIQHRFTKTAKNVIERRYPKSKLEMDEAKRKYKWGRYGRGK
YVYPDEQATALREFITEQIFDKFPNAKIEYFT
>lcl|NZ_CP026520.1_prot_WP_042228664.1_2132 [gene=mntR] [locus_tag=PC41400_RS10945] [protein=transcriptional regulator MntR] [protein_id=WP_042228664.1] [location=2396239..2396673] [gbkey=CDS]
MATPSMEDYLERIYKLIDEKGYARVSDIAEGLEVHPSSVTKMIQKLDKDKYLVYEKYRGLVLTSKGKKMGKRLVDRHALL
EQFLTVIGVQEENIYRDVEGIEHHLSWDSITCIETLIEYLNRDPARLEDLRNLRTELDNSSESE
>lcl|NZ_CP026520.1_prot_WP_042228662.1_2133 [locus_tag=PC41400_RS10950] [protein=patatin-like phospholipase family protein] [protein_id=WP_042228662.1] [location=2396793..2397758] [gbkey=CDS]
MKINAVFEGGGVKGIALAGAVSSALDNGFEFHDVAGTSSGSIVAAFLAAGYTGEEIKRLIIETPFHKFLKRSPIFDMGIV
GSAARLIWKKGLYSGDALEDWVRQKLEVKGIRKFGDLRKNQLRIVASDISQGKLLVFPDDVAQYGIDPGRFSIARAVRMS
SSIPYFFDPVKIRKPARQTRHDEPFRNQFVYIVDGAILSNFPLWLFDEPEQQPSRIPTLGFSLVGKIPKHIHKINGVISM
FQALFTTMMDAHDTRYIEENNYFRTVKIPALNVNSTQFNLTLEDSENLYEAGYRASDDYFRKWSMHLYESAYEKYVLRKP
V
>lcl|NZ_CP026520.1_prot_WP_042228659.1_2134 [locus_tag=PC41400_RS10955] [protein=hypothetical protein] [protein_id=WP_042228659.1] [location=complement(2397952..2398239)] [gbkey=CDS]
MNRRWSAATVVIAALALIGLAYSLNQFLIPVLVLGLIFVLYKMAPRKGQGTSYKTRSAGRKPDRLSVSERRRKAKFRVIE
GSKPSDPPEDPPRYH
>lcl|NZ_CP026520.1_prot_WP_042228658.1_2135 [locus_tag=PC41400_RS10960] [protein=DUF1385 domain-containing protein] [protein_id=WP_042228658.1] [location=complement(2398236..2399186)] [gbkey=CDS]
MSQSNTSIYGGQAVIEGVMFAGRNVHVTAVRKKDRTIDFLEVPKTEIGWVNKLKKIPLLRGIVGIIESSAKGAQHLNFSA
ESFAEEEEGEGKAPAKEEKSGFSLSMILGVAVVGVLSFLMGKVIFTLVPAILEDVMFGKFFEQHQIWHNIIEGLIKIVLL
VSYIYIISLTPVIKRLFQYHGAEHKVISAYEAGVELNVRNVQQFSTLHYRCGSSFIVFTVIIGVIIYSFFTYDNMIDRIV
QRLVLLPVIIGVSYEVLRFTNSLREVPVLRFLGYPGLWLQKLTTKEPTDDQVEVSIASFNRMREAESRLTQSKASA
>lcl|NZ_CP026520.1_prot_WP_042228656.1_2136 [locus_tag=PC41400_RS10965] [protein=YqhR family membrane protein] [protein_id=WP_042228656.1] [location=2399346..2399903] [gbkey=CDS]
MVNHTQGTAAAGNGNSACGERKSFSQNGGAAGEEKKTNRAFFALYLGFFAGIIWGAVKIIEYIFKFTRVIPAFLAEPFYR
HDFLLTWQGTLVGWLFFIVFSILASFIYMFLFGKVRGPWMGVGYGIAWWCLLYLVIGPWTGMMGSMYFSDWNTILTDFCL
FLVWGLFIGYSTAFEFTEEHSRKPA
>lcl|NZ_CP026520.1_prot_WP_042228655.1_2137 [gene=aroQ] [locus_tag=PC41400_RS10970] [protein=type II 3-dehydroquinate dehydratase] [protein_id=WP_042228655.1] [location=2399998..2400441] [gbkey=CDS]
MNKVLVLNGPNLNMLGIREPAVYGSMSLKDIETAMTEIAGTLDLQLTFHQSNHEGQLIDWIHEAHGTMDGILINPGAFTH
YSYAIRDAISSVGLPAVEVHLSNIHKREEFRHRSVLAAVMVGQIAGFGAFGYELGLRALHQHFVEAR
>lcl|NZ_CP026520.1_prot_WP_042228654.1_2138 [locus_tag=PC41400_RS10975] [protein=Xaa-Pro peptidase family protein] [protein_id=WP_042228654.1] [location=2400446..2401516] [gbkey=CDS]
MQDKRLERLRRQLTEQGIPAVLVTNPINRRYLTGFTGSAGIVLVTGGRAVLITDFRYMEQAPKEAAGYEVIDHKGKIGRT
ILDILGKESIKQLGFEENDLTYSAYSGYTGDLQGVELIPTGGIVEKLRIVKDESELKIMQEAADLADRTFSYILTRLRTG
VTEKQIALEMEIFIRENGGTSSSFDTIVASGERSALPHGVASDRVLQPNEFVKLDFGALYKGYCSDITRTVVLGQPSDRH
KEIYGIVLEAQLNCVGSLKPGMTGSEGDALCRDIIARYGYGDNFGHSTGHGLGMEVHEHPRLAKGFAGVLEPGMVVTVEP
GIYIPGFGGVRIEDDVVIEQSGIRLLTHSTKELITI
>lcl|NZ_CP026520.1_prot_WP_042228653.1_2139 [gene=efp] [locus_tag=PC41400_RS10980] [protein=elongation factor P] [protein_id=WP_042228653.1] [location=2401543..2402100] [gbkey=CDS]
MISVNDFKTGLTIVVDNDLFSVIDFQHVKPGKGAAFVRSKLKNLRNGNVVEKTFRAGENVGRAHIENREMQYLYASGSEY
TFMDTETYDQISLEEAQLKWELNFLKENMNIHIMSYQGEILGINLPNSVELKVVETEPGVKGNTAQGATKNAKVETGLNV
QVPLFINENDVLLIDTREGKYISRA
>lcl|NZ_CP026520.1_prot_WP_042228652.1_2140 [locus_tag=PC41400_RS10985] [protein=ABC transporter substrate-binding protein] [protein_id=WP_042228652.1] [location=2402326..2403357] [gbkey=CDS]
MKMKPFVTLLLAASLVTAAGCGKQAAPESTGGGASAAPEGKKVKIGIMQIVEHPSLDAARQGFIAALKDNGYEDKKTADI
DYQNAQNDMNNNTSIAQKFAADKKDLVLGIATPSALAAAQTVKDAPVLFTAVTDPVDAKLVKSIEKPGGNVTGTSDLDPK
AVDQLTEFVGKHFPNVKTIGIVANEGERNSQVQIKQATEAFAKHNIKVVKASPSNSSEVKQAAESLIGKADALYVPLDNT
VVSALNAVIQVAEQGKKPLFVGENDSVKGGGFAGYGSNYYDLGYTTGLQAVEILKNGKKPSDIPVGYPKKLDLAINMKAA
KNMGVEVTDAMKAEVKEKANLFE
>lcl|NZ_CP026520.1_prot_WP_042228651.1_2141 [locus_tag=PC41400_RS10990] [protein=ABC transporter permease] [protein_id=WP_042228651.1] [location=2403422..2404345] [gbkey=CDS]
MMTIDWGTVLQGSLEQGLFYAIMALGVYITFRILDFPDLTVDGSFTTGASIAAVMITHGVSPLLATLAAFAGGAVAGLIT
GLLHTKGKINALLSGILMMIALYSINLRIMGKATISLFGSEVLLDKESLLYIMPIVVVAIKLALDWFLHTELGLALRATG
DNKRMIRSFGVNTDNTIILGVSLSNGLVALSGALIAQYSGFADMQMGVGMIVVGLASVIIGEALFGAKTIIRATFAVVLG
AIIYRFVVAIALQVGLESTDMKLMTALIVIIALTLPKIQGSLKQKAVIRKRKTELAEKSGGVEHASH
>lcl|NZ_CP026520.1_prot_WP_042228649.1_2142 [locus_tag=PC41400_RS10995] [protein=ABC transporter ATP-binding protein] [protein_id=WP_042228649.1] [location=2404332..2405126] [gbkey=CDS]
MLHIDRASKIFNPATPDEKVALSDIQLHLAPGDFVTVIGSNGAGKSTLMNLISGVMQPDYGVVRIEGKDITSLSEHKRSR
WIGRVFQDPMAGTAPTMSIQENLAMAHARAGKRGFSLGVTKQKKLAFKEHLERLGIGLENRLNAKVGLLSGGERQALSLL
MATFTQPNILLLDEHTAALDPSRAELITQLTKQIVEESKLTTLMVTHNMEQAIRLGNRLIMMDKGRIILDIPAASKQDLT
VEKLLREFEKIRGEKFVDDRVVLG
>lcl|NZ_CP026520.1_prot_WP_042228648.1_2143 [locus_tag=PC41400_RS11000] [protein=aspartate kinase] [protein_id=WP_042228648.1] [location=2405330..2406586] [gbkey=CDS]
MSLIVMKFGGSSVGDAERMKRVAKRIVDRKREGHRCVVVVSAMGDTTDDLIDLSKQLYDGAPPAREMDMLLTTGEQVSVA
LLSMAIQNMGEPAVSYTGWQSGMITEDLHGKARITDIQPHRIHTALEKDNVVIVAGFQGMTEDGEITTLGRGGSDTTAVA
LAAAIHADLCEIYTDVDGIYTTDPRVVKNARKLKEISYDEMLELANLGAAVLHPRAVEYAKNYNVALVVRSSFTQNEGTY
VKEEAEMEQGIVVRGIAYDKNIARISILGVPDKPGQLAKVFTSLAAEQIDVDIIVQSGVLNGLADFSFSVSLSDRDKALQ
TIEGIRGEVGFREVTSEGDLVKISIVGAGMVSTPGVAAKMFDELSKIGVSINMVSTSEIKISCVIDGKQLAEAVQALHTA
YGLDAQEQVFVGGPQDRR
>lcl|NZ_CP026520.1_prot_WP_009674889.1_2144 [locus_tag=PC41400_RS11005] [protein=DUF441 domain-containing protein] [protein_id=WP_009674889.1] [location=complement(2406753..2407217)] [gbkey=CDS]
MNGDLLLVILIIVGLIGRSPIITTAACVLLVVKLISLDRYLPAIERRGLELGLLFLTMGVLVPFASERISYKDILTVFTT
WPGILALSGGAIATYMNGKGLDLLKIDPQLIVGLVIGSIFGIIFMRGIPVGPLMAAGITAVLLKIFTFVADRIS
>lcl|NZ_CP026520.1_prot_WP_042228647.1_2145 [locus_tag=PC41400_RS11010] [protein=2-phosphosulfolactate phosphatase] [protein_id=WP_042228647.1] [location=complement(2407311..2408048)] [gbkey=CDS]
MRIDVIPSLSEVRSELLAGRTAIVIDVLRATTTIVSALESGSAGVLAAESVQQAVMLRGKRDVLGGERRCRRIPGFDLGN
SPLEYTPEAVGGRRVVLTTTNGTRAMQKAAPADRIVAAALLNGEACARAAAGWKRDVALLCAGTAGAFCFEDGLCAGLVV
SALQRLCGGTALVCDLGRSLAACYERHADDPATPLAACDSGRRLSSIGCETDVAFAARVNVSDVVPLLKDGILVPLAPAR
VHLSS
>lcl|NZ_CP026520.1_prot_WP_042228646.1_2146 [locus_tag=PC41400_RS11015] [protein=phosphosulfolactate synthase] [protein_id=WP_042228646.1] [location=complement(2408038..2408883)] [gbkey=CDS]
MEESSCKKWHPTLADPTGCRQSKPRALGKTMVMDKGLGLNAYDDLLQAAGEHVDYIKLGFGTSPLYPRELLRHKIERAKA
LGIDVYPGGTFLEVAVQEGLISSYFDTVAALGFTCVEISDGTIELDRSLRSELIVRGIEAGLKVITEYGKKCWGSSVEAE
ALVATVTGDVEMGAELVTIEARESGMGVGIFDENGACKDEELNGVLRSVPSPDILLWEAPLKNQQIHLLELLGPDVHLGN
VAPADVIALEALRRGLRSDTFHFGERLRKSARKEGIEVHAD
>lcl|NZ_CP026520.1_prot_WP_042228644.1_2147 [locus_tag=PC41400_RS11020] [protein=YqhV family protein] [protein_id=WP_042228644.1] [location=2409030..2409296] [gbkey=CDS]
MINKIVLSMASLRLLSGSLEIIAALLMIRFNQIEKALLVNTGLAMAGPLILLATTTIGLVGIAEKLSIGKMLWVVAGVSC
IFIGILKK
>lcl|NZ_CP026520.1_prot_WP_042228643.1_2148 [locus_tag=PC41400_RS11025] [protein=stage III sporulation protein AA] [protein_id=WP_042228643.1] [location=2409378..2410601] [gbkey=CDS]
MLASVFAHLPASIRGILESLPVNLHSGLEEIRVRQDRPLEIVFDRRCAFVTREGLTDADASRAYRPSREECKSLLELLTR
HSFYSFEEELKRGFITIPGGHRVGLAGRVVLEGGSVKHIRDVSSFNIRIARELPGAGKPVLPHLLDREADTVHHTLLISP
PQLGKTTLIRDLARMISSGEAAFPAAGGEGSPGGAGASPSGGSRTGTSGGPGLPGGTGSTGISPAGAGPEAGSREPVTRP
TARHSPRRSAGWKVGIVDERSEIAASERGVPRFDLGPRTDVLDGCPKAEGMMMMIRSMSPEVLIVDEIGRQEDAEAIREA
LHAGIRIIATAHGRDLGDVLRRPALRELLESGVFGRYVILGRSAAGGRRVQVLDGEGRQLAVFGAYRAPAEEKPSGWETI
FSTGGRG
>lcl|NZ_CP026520.1_prot_WP_042228642.1_2149 [gene=spoIIIAB] [locus_tag=PC41400_RS11030] [protein=stage III sporulation protein SpoIIIAB] [protein_id=WP_042228642.1] [location=2410606..2411127] [gbkey=CDS]
MWVKLAGAALVLGAGAGFGFAKARRYRRRPQQIREVIQALQRLETEIVYGSTPLPDAIATLSEQTRAPLAGVFREVADRL
SEPQDDTLSAVWRDAVQHGWRNSAMKPAELQVVLRLGQTLGASDRSDQAKHLRLAVLQLQAEETTAAEEQKRYEAMWKSL
GVLVGALIVILMY
>lcl|NZ_CP026520.1_prot_WP_009674926.1_2150 [gene=spoIIIAC] [locus_tag=PC41400_RS11035] [protein=stage III sporulation protein AC] [protein_id=WP_009674926.1] [location=2411143..2411346] [gbkey=CDS]
MNVDVNAIFQIAGIGIIIAMIHTVLKQMGKEDMAHWVTVIGFVVVLFMVVRLLDNLFQEIKTIFLFQ
>lcl|NZ_CP026520.1_prot_WP_042228641.1_2151 [gene=spoIIIAD] [locus_tag=PC41400_RS11040] [protein=stage III sporulation protein AD] [protein_id=WP_042228641.1] [location=2411357..2411749] [gbkey=CDS]
MEIIQIVGLGLIATVLILIIKEQKPMFAFLLAAFTGILIFLFLIGKIASVIEVLNNLAQQSGVNMVFLKTILKIIGIAYI
AEFGAQIVRDAGQESIASKIELSGKILIMVMAIPIITVIIETVIKLFPAT
>lcl|NZ_CP026520.1_prot_WP_042228640.1_2152 [gene=spoIIIAE] [locus_tag=PC41400_RS11045] [protein=stage III sporulation protein AE] [protein_id=WP_042228640.1] [location=2411801..2413000] [gbkey=CDS]
MAQQLSHRTWTTLVLFVALFLGFLSHTGAQKLNPVNPIDTLVKEQTNQLDTESVEKFWDKLMKEYGGYFPDNRPPSFSDL
ITGTKTFDFSNLWKGALKYLFHEIVVNGKLLVSIVILTVFSMLLETLQSSFERNNVSKVAFVVSYLVLLIIAVNSFQVAI
EYAKGAISSMIQFMIALMPLLLSLLASSGGITSVAILHPLIVFMIHTIGTVIYVFVFPLLFFSAVLHIVSSMTDKYKVNR
LANLLQKIAIGTLGVFVTVFLGVISVQGSMGAVTDGVTLRTAKYIAGNFIPIVGRLFSDATETVLGASLLVKNAVGLTGV
VIVILLCAFPAIKILVLALIYHVCAAVMQPLGDSPIIVCLSTIGKSMVYVFAALSAVGLMFFLALTIMITAGNIAVMVR
>lcl|NZ_CP026520.1_prot_WP_241688841.1_2153 [gene=spoIIIAF] [locus_tag=PC41400_RS11050] [protein=stage III sporulation protein AF] [protein_id=WP_241688841.1] [location=2413143..2413928] [gbkey=CDS]
MEWLGGWLKTVVALILLATFIELMLPTNKMQRYVRTAMSLFILLTLLSPVLQLARKSWDPARLLAEAQSGQQLGGGGAAD
GRQAPAGTGPSGQPASLQTVMEQADRLQAGSREQAVRLLEEKLAAEMRQTLQKETPYPIRDIRVKTAVDGQGAASVASVE
LTLDDAASPAAEPDAYGDALAGEAGASGMTPVAPIAPVAPVDVRIGGASGPGGRAVPAQTGEQPSPAQREAERKLAKAAE
LGWKVDPSHISVQWTKEQSKL
>lcl|NZ_CP026520.1_prot_WP_042228637.1_2154 [gene=spoIIIAG] [locus_tag=PC41400_RS11055] [protein=stage III sporulation protein AG] [protein_id=WP_042228637.1] [location=2413972..2414613] [gbkey=CDS]
MGSWKQWRSKWFGTGEDGKRSNSFLWIVLLGLVGAALMILSSFLNFKEVEPAGDSRASPSTQQGKETFMGGAAKDKTPFR
EYEEAYESRLRDILRKVVGVGDVEVLITIESTEEIVVEKDSKENQQLTTEKDTNGANRNISQVTKEGKVVLYQVSGDQQP
LVVKTIKPKVSGVLVVAKGAEDLTVKKMMIEAVARGIDVPVHRISILPRKTAE
>lcl|NZ_CP026520.1_prot_WP_042228636.1_2155 [locus_tag=PC41400_RS11060] [protein=SpoIIIAH-like family protein] [protein_id=WP_042228636.1] [location=2414710..2415519] [gbkey=CDS]
MNAKRQTIWLVSMLSLMVVLSAYYLFTEDVSKVDVASTAPKADEVKIDLSQVSPEQAVADAGKASADKAADAKADGTKTT
ADAKTDAKTDAKTDAKTDAKTDTKTDAKAEAKTGDAKAQDSKTGATASKDGKDAAAAPASAEAKISQQTQASSKSGRDYL
TKLQLDRQDALSKKTEELLTLITDKNQSSDAVSKAQQELQQIEDTNTRVTNVEDVLSKDYPNAVLTQDGSKWKVTVQSDK
LEKSQAVSIVETVMKELKVGPEISVQYVK
>lcl|NZ_CP026520.1_prot_WP_042228634.1_2156 [gene=accB] [locus_tag=PC41400_RS11065] [protein=acetyl-CoA carboxylase biotin carboxyl carrier protein] [protein_id=WP_042228634.1] [location=2415678..2416172] [gbkey=CDS]
MFKLSEIKELMKLMDQTSLQELEIENEGSRLTIRKPNKTESVVVTAAPVQQAYVPAIAQAVPGSAPAQVAAEAPKADQQK
ESDSTLHKIVSPMVGTFYQASSPGSDPFVSKGSNVSEKSVVCIVEAMKLMNEIEAEVKGEIVEILVENGQLVEYGQPLFL
VKPE
>lcl|NZ_CP026520.1_prot_WP_042228633.1_2157 [gene=accC] [locus_tag=PC41400_RS11070] [protein=acetyl-CoA carboxylase biotin carboxylase subunit] [protein_id=WP_042228633.1] [location=2416191..2417549] [gbkey=CDS]
MGFQKVLIANRGEIAVRIIRACRELGIQTIAVYSEADRDALHVRLADEAYCIGPTPSKDSYLNFTNIMSVATLTDTDAIH
PGYGFLAENADFAEICESCNITFIGPSPEAITKMGDKAVAKQTMKDAGVPVIPGSDGLIEDLDDAISLGRDIGYPLIIKA
TAGGGGKGIRIAESEDMLIQHIMTAQQEAQLNFGNSGVYLEKYLTGMKHVEIQIIADKHGNAVHLGERDCSVQRRRQKLV
EEAPCPTLTPEIRQAMGEAAVRAAKAVNYSGAGTLEFLLGPDGNFYFMEMNTRIQVEHPVTEFVTGVDLIKEMIRVAEGN
PLPFTQEDIQINGWSIECRINAEDPERNFMPSPGRINFYLPPGGIGVRVDSAAYPGYVIPPHYDSMIAKLIVWGATREEA
VNRMKRALAEFEVDGVHTTIPFHLKLLNHRKFLSGDFDIKFLEEHDVNDPES
>lcl|NZ_CP026520.1_prot_WP_042228632.1_2158 [locus_tag=PC41400_RS11075] [protein=Asp23/Gls24 family envelope stress response protein] [protein_id=WP_042228632.1] [location=2417635..2418045] [gbkey=CDS]
MTIIAADYEKTDMGTIQIAPEVIEVIAGLATVEVEGVAGMSGGFVNGITELLGRKNFAKGIKVNVGQREAAIDVSIVMEF
GHRIPEVAGKIQQNVKLAIESMTGLSVVEVNVHIHDVQFKTAEKPVEEDSAALRVK
>lcl|NZ_CP026520.1_prot_WP_042228630.1_2159 [gene=amaP] [locus_tag=PC41400_RS11080] [protein=alkaline shock response membrane anchor protein AmaP] [protein_id=WP_042228630.1] [location=2418125..2418691] [gbkey=CDS]
MVKLLDRLLLFLFSLAIMFLSAIAVLTAFQLIPMRSTDYFLQDVFDSGFTVSKAWVIGAGIVIFLISIRFFYISVRRSRT
QAPSIDQRSEYGDIRISVETVENLALKAAGRSRGVRDLRARVKVSEAGLEIGIRTVVDGESSIPELTEEIQSAIKNHVEE
ITGIPVSGVSVYVANIVQNTPTFKSRVE
>lcl|NZ_CP026520.1_prot_WP_042228629.1_2160 [locus_tag=PC41400_RS11085] [protein=DUF2273 domain-containing protein] [protein_id=WP_042228629.1] [location=2418706..2418936] [gbkey=CDS]
MWRKLWQEHPGTITGGGAGLVLGLVYLFFGFWNMLIFAFIVFLGYYIGKKVDRGEDLLLPFQNLWQFLTQKWRMFR
>lcl|NZ_CP026520.1_prot_WP_042228627.1_2161 [gene=nusB] [locus_tag=PC41400_RS11090] [protein=transcription antitermination factor NusB] [protein_id=WP_042228627.1] [location=2419041..2419502] [gbkey=CDS]
MRRRVAREIALQSLYQIDMNGVSAAEAVNVAVHEAENDNEAQVDHKEKIMPEYIRELVDGTLNNRDRIDQLLAGYLKGWK
MDRLSRVDREVLRLAAFEMLFQEDVPPKVVVNEAIDLAKHFGTDDSGKFVNGVLGKMIKDVDTLREHIRSGKE
>lcl|NZ_CP026520.1_prot_WP_042228626.1_2162 [gene=folD] [locus_tag=PC41400_RS11095] [protein=bifunctional methylenetetrahydrofolate dehydrogenase/methenyltetrahydrofolate cyclohydrolase FolD] [protein_id=WP_042228626.1] [location=2419651..2420523] [gbkey=CDS]
MAAEVINGKEIVRSIREEIKQEVEELSARGTTPGLVVIIVGDDQASHVYVNNKEKACAQVGIQSHVHRLPADTIQEELVK
LIHEYNRNPEFHGILVQSPLPGHINEEQVVDEITPEKDVDCFHPVNVGNLMIGKEGMLPCTPAGVMEVLKKIGVSPAGKH
AVVVGRSNIVGKPMAMLLLREHATVTICHSRTPNLQDITRQADILVAAVGKAGVIEDRHVKPGAVVIDVGMNRNAEGKLC
GDVDFDAVKEIAGYITPVPGCVGPMTITMLLKNTLSSARRASNTGTEAGV
>lcl|NZ_CP026520.1_prot_WP_042228625.1_2163 [gene=xseA] [locus_tag=PC41400_RS11100] [protein=exodeoxyribonuclease VII large subunit] [protein_id=WP_042228625.1] [location=2420527..2421894] [gbkey=CDS]
MSERAVLSIKDLNRYIKMKLEGDNRLQDVWVRGEISNFTHHSSGHMYFTLKDKDSRLKSIMFASYNQRLPFRPKEGTMVI
ARGNVSVYERDGQYQFYVTQMQPDGIGGLYLAFEQLKQKLDSEGLFRTERKKPIPRMPKAVGVITSPTGAAVRDIIITLQ
RRFPNVPILLHPVLVQGAQAAPSIVRAIETMNNLGEVDVLIVGRGGGSLEELWAFNEEIVARSIFASAIPIISAVGHETD
FTIADFVADLRAATPTAAAELAVPHHLELKQQMAHLGQRLHRGLLQQLQRRQDRLERLRRSPYLTNPHRQLLQHPAQRLD
RLGEQLSFRMRQLVGRAGDRRMRLERTLSGFNPQEQVRYARKRVEGAERALTLAMQGVARGRRQQLVSTLRHLDALSPLK
VMQRGYSLVYDEKEKRVLNSIEQVQIGDMVRVQLTDGRLDCHVWAMEEKTNGERK
>lcl|NZ_CP026520.1_prot_WP_042228624.1_2164 [gene=xseB] [locus_tag=PC41400_RS11105] [protein=exodeoxyribonuclease VII small subunit] [protein_id=WP_042228624.1] [location=2421878..2422150] [gbkey=CDS]
MGNESNQEETAISFEEAMDKLEGIVAKLESGDVPLEQAIELYQEGMKLSHLCGQKLETVEKKIEMLVESEAGLSKKPFGA
NMIEDKGESR
>lcl|NZ_CP026520.1_prot_WP_241688889.1_2165 [locus_tag=PC41400_RS11110] [protein=polyprenyl synthetase family protein] [protein_id=WP_241688889.1] [location=2422159..2423049] [gbkey=CDS]
MNIQEYIASYAKEVEGELQELFPRLWDIPDVLRQSAQYSIDAGGKRLRPILVLAAAEALGGRKEAAMPVACAIEMVHTYS
LIHDDLPAMDNDDYRRGKLTNHKVYGEAMAILAGDALLTQAFYSASQARRHGVPAEQVLNIVEELAEYSGARGMVGGQVA
DMKGEQGITELSGLEYIHQHKTSDLIMFSLRAGGRVAGADDKQLEALTDFGRSIGLAFQIQDDILDVTGTQEKLGKAVGS
DEKQHKVTYPFFIGMEESERRVAQLTDEAKTAIAGGVLPKPQRLLELADYLMKRDR
>lcl|NZ_CP026520.1_prot_WP_042228622.1_2166 [gene=dxs] [locus_tag=PC41400_RS11115] [protein=1-deoxy-D-xylulose-5-phosphate synthase] [protein_id=WP_042228622.1] [location=2423144..2425045] [gbkey=CDS]
MLLDHINQPEDLKRLSTAELEQLAEEIRQFLVEKLSASGGHLSSNLGIVELTIALHTLYDSPRDKFIFDVGHQAYVHKIL
TGRKDRFDTLRKYKGMCGFIKRAESDHDVWEAGHSSTSLSAAMGMAMARDLKGEDFRVFAVIGDGAMTGGMALEAMNHIG
HEKKNLTVILNDNEMSIAPNVGAIHNYLSKIRSDKHYRKAKEEVESLLKKIPAIGGTLAKTAEKVKDSLKYLLVSGVWFE
ELGFNYIGPVDGHNLPLLLETLKQSEKVEGPVLIHVITTKGKGYSPAEMDSHKWHGASPYKIESGQFLKAVGNPMYTEVF
GNTLIEFAEKDERVVAVTPAMPGGSGLMNFAARFPERMIDVGIAEQHAATLCAGMAAEGMKPVFAVYSTFLQRAYDQVVH
DICRPKLNVMFAIDRAGFVGPDGETHQGVYDISFLRSLPNMVLMMPKDENELRRMMRTGIDYNEGPIAVRYARINGLGVP
MDENPQPIPIGTWETLTEGETMAVLAVGPMNQVALEAADMLRKEGINLRVINARFIKPMDEAMLLQLAKEHMPIITMEEG
SEQGGFGSAVLEFYAKQGIEGMRVRTMGVPDYFVEHGSIKEQRQEVGLTAEHLITQVHTLLPRKRQRSAGLNA
>lcl|NZ_CP026520.1_prot_WP_042228621.1_2167 [locus_tag=PC41400_RS11120] [protein=TlyA family RNA methyltransferase] [protein_id=WP_042228621.1] [location=2425173..2426015] [gbkey=CDS]
MATDKERIDVLLVEQGYFESREKAKAAIMAGLVFADEEPVDKAGMKISRAAALKVKGAVHPYVSRGGLKLEKALKQFNID
VKNAVMLDIGASTGGFTDCALQNGASYVYAIDVGYNQLDWSLRQDERVQVMERTNFRYLTPEELTGPPPSFATIDVSFIS
LKIILPPLKTMLGVKGETVALIKPQFEAGREKVGKSGVIRDSEVHKDVLESVLGFAKEIGFYLKGLTFSPITGGEGNIEF
LAYWTCNPEDTEKLEEPLQSLIDRVVPEAKAMFAKPETRY
>lcl|NZ_CP026520.1_prot_WP_042228620.1_2168 [locus_tag=PC41400_RS11125] [protein=hypothetical protein] [protein_id=WP_042228620.1] [location=2426145..2426615] [gbkey=CDS]
MQHGDTLILILLLGLLAAWAGISFRKWLYEPYADSFQPNRITDYNEVPEDDDVVELLIGAGFDVVGGKTKIPISMTLNQK
EELYSRLFIDYFAEKEDNLYLVKVARERKPLEMTGSAIRDMLLVYSLLVPEAAGVLYVDMNGRKIKQITFHIEVFE
>lcl|NZ_CP026520.1_prot_WP_042228619.1_2169 [gene=argR] [locus_tag=PC41400_RS11130] [protein=transcriptional regulator ArgR] [protein_id=WP_042228619.1] [location=2426612..2427058] [gbkey=CDS]
MKGQRHVKIREIITGQEIETQDELVESLRTAGFNVTQATVSRDIKELHLIKVPLDDGRYKYSMPADQRYNPMQRLKRALS
DHFINIDFTENLVVMKCMPGTANTICALIDNLEWTEIMGTICGDDTILIICRTKDVSQKVVNQIMSLI
>lcl|NZ_CP026520.1_prot_WP_042228618.1_2170 [gene=recN] [locus_tag=PC41400_RS11135] [protein=DNA repair protein RecN] [protein_id=WP_042228618.1] [location=2427091..2428827] [gbkey=CDS]
MLTELSIRNLAVIEYVHIGFKSGFHVLTGETGAGKSIIIDALSLIVGGRGSSELVRYGSEKAEIEAVFEIGEGHPVWTTL
AQLGIEAEGEEQLLIRREVTAGGKSSSRINGQLVNLTMLREVGGWLVNIHGQHEHQSLLKSDEHIHWLDAYGDKEVGPNL
AAYRESYDRYTKLRKELRALQETSKQSLQMIDLYRFQVDEIAAASLKEGEDVTLAEEKSKLANAEKLYSSVAASYDTLYG
ASNGLEAIGKSIARIQDVAAIDSAGIAPLLEQMQSAFYQLEDAAYQLRDYRDDIEFNPERLDEIELRLDMIASLRRKYGD
TIAEILAYSEKVSGELATIENKDEIIQKLQEKLDREIVQLTKAARKLSDSRKKIADRLAGEIETELRDLHMERTQFRVQV
SHTPVPAGGGGLEIGDQPVKFTRDGIDLVEFLMAPNPGEPLRGLSKIASGGELSRIMLAMKSIFARLDQIPVLVFDEVDT
GVSGRAAQAIAEKLSLLSRSCQVFSITHLPQVACMADVHFYIHKEVQGERTFTRVTDLDRRGRIEELARMLGGVEVTETT
LHHAQEMLALADEKKSRN
>lcl|NZ_CP026520.1_prot_WP_042228617.1_2171 [gene=spoIVB] [locus_tag=PC41400_RS11140] [protein=SpoIVB peptidase] [protein_id=WP_042228617.1] [location=2428939..2430273] [gbkey=CDS]
MKSNRRRRFIGLLLVFFVCILSVSAPFQSFASFPDELRLFSGQAKELRLSMPVNAMLTVEDPEIVKVNGNSRHSFQVNLK
DPISLQSNQQGYTEMKLKLFGAIPLKTVKVNVVPDLKVLPGGQTIGVKIKSAGILVVGHHLVAVTEKQKISPGEEAKVQV
GDLIVAINGSPSADVSKVSGLVNKAGQSKKPLELTVQRGQQRFELKLKPAYDVVDKSYRLGLYIRDSAAGVGTLTFYAPE
QNVYGALGHVITDMDTQTPIVVGSGEVVHSNVTSIAKSQNGEPGEKHAQIFRDSKVIGTIEKNTPFGIFGKMEQAPEYSL
SRTPIPVAFAEEVKEGPAQIYTVVSGQKVEKFDIEVVHVAKQGNPATKGMVIKVTDKRLLEKTGGIVQGMSGSPIIQNGK
MIGAVTHVFVNDPSSGYGCFIEWMLQDAGILLRPESDAKGLKAS
>lcl|NZ_CP026520.1_prot_WP_042228616.1_2172 [gene=spo0A] [locus_tag=PC41400_RS11145] [protein=sporulation transcription factor Spo0A] [protein_id=WP_042228616.1] [location=2430470..2431264] [gbkey=CDS]
MQKIQVLLADDNREFTHLLSEFISEQEDMEIAGIAYNGNEVLRFMESQKDVPDILILDIIMPHLDGLGVLEKLREMNLNP
APRIIMLTAFGQENITQKAVQLGASYYILKPFDMEVLTNRIRQLVGNQPAITSTSSIKANVVQMPKSKNLDANITTIIHE
IGVPAHIKGYQYLREAITMVYNNIEILGAITKTLYPAIAEKYKTTPSRVERAIRHAIEVAWTRGNIDSISFIFGYTVNIS
KSKPTNSEFIAMVADKLRIEHKVS
>lcl|NZ_CP026520.1_prot_WP_042228615.1_2173 [locus_tag=PC41400_RS11150] [protein=tyrosine-type recombinase/integrase] [protein_id=WP_042228615.1] [location=2431428..2432411] [gbkey=CDS]
MKMTDLEFYVEDFLTFCQNKNLSRKTILSYDQTLKLFLAYLKNEHGVEKITEVRTGHIRQYVAYVQERGKYTVVNRESSK
LVNFPENRTDYKKPISTITVNNYIRNIKVLFNWLQQEGELVKNPVDNISLIKTVRRQKKGINQDEFNKLLEQFDYTTFHG
YRNKIIVLLLQDTGMRIGECLELGVDAIDFKHKMILLTKTKGNKERYVYFSNVMGRELKQYLKHRDRYTETSLLFPTTKG
TTLSVQSFEKQLRDAGRNAGVEVHPHQIRNNFARQYLLNGGDFYTLSRILGHSSVTVTEQAYMDLTREEIAKKYQNHSPL
GRWKIQG
>lcl|NZ_CP026520.1_prot_PC41400_RS11155_2174 [locus_tag=PC41400_RS11155] [protein=recombinase family protein] [pseudo=true] [partial=5'] [location=<2432711..2433118] [gbkey=CDS]
YLREGDSLVVTKLDRLARSTFDLHRIAKSLEERGIHLVILKQQIDTSTPAGKLMFTMLGAVAEFEKDLINERTAEGRARA
KAKGTHMGRKGQDEKQVKKALSLYAARDENGLSVNEISNMTGVPRSTIYAKIKEQ
>lcl|NZ_CP026520.1_prot_WP_129112473.1_2175 [locus_tag=PC41400_RS11160] [protein=hypothetical protein] [protein_id=WP_129112473.1] [location=complement(2433744..2433962)] [gbkey=CDS]
MAKIATDVVVIEFVVVAGVRRIQEFRVIGSAKPCQGFLAVGKRVRAAERCLLTHGFKLLASSRDVLVFTRPS
>lcl|NZ_CP026520.1_prot_WP_042228614.1_2176 [locus_tag=PC41400_RS11165] [protein=succinylglutamate desuccinylase/aspartoacylase family protein] [protein_id=WP_042228614.1] [location=complement(2434061..2434765)] [gbkey=CDS]
MQITKHQLAKGTKFETPYYIIKGNTAGKTVMITAGVHGDETANVLAAKKLVDLLQKNVHHIDKGTLIIVPIVNQIAYKRR
FRGIPDLNRTFPCKPNVAARHPLASELFRLAKQFQPEWYIDLHEARGFSKIDPSVLGQTLLTNPRSEAIPLVKRIVNHLN
RSIAQKSRQFSIRIRHLPGSGRTAAYRLLKSNSITVETSTSMPKAIRVEYQMKILRYLLNETGLIKDERTSATA
>lcl|NZ_CP026520.1_prot_WP_053228683.1_2177 [locus_tag=PC41400_RS11170] [protein=YheC/YheD family protein] [protein_id=WP_053228683.1] [location=2434908..2435642] [gbkey=CDS]
MGGLITKNNVSKSLQTNELLSGNVPETKIYSESMLKTMLNKYSMVYVKPNRGTGGKGIFRVEKLGAGSYKYQINTMTKTF
ESFDSMAKSINIKTKSEKYAIQHGIHLMKHKNRLFDLRIMVQKNPNGKWETTGVIGRLGHPKKIVTNVCKGGSSKPIEVL
LKDHVTNIPEYKKWLENLGYRAAKQLNKTFPRIKELGLDVGIDCKLHPWILEANPQPAIYGFKTLKDKSIYQKICRYQKA
YGRQ
>lcl|NZ_CP026520.1_prot_PC41400_RS11175_2178 [locus_tag=PC41400_RS11175] [protein=sporulation initiation factor Spo0A C-terminal domain-containing protein] [pseudo=true] [partial=5'] [location=<2436141..2436269] [gbkey=CDS]
RGNIDSISHLFGYTINIAKSKPTNSEFIAMVTDKLRIEHKVS
>lcl|NZ_CP026520.1_prot_WP_042228613.1_2179 [locus_tag=PC41400_RS11180] [protein=Ig-like domain-containing protein] [protein_id=WP_042228613.1] [location=2436619..2437677] [gbkey=CDS]
MTSMSKRISVFLFFCFMMQALSFGTAVHASELQPDLVSNGGFESVKSGAGKWEAGVEPAGWGLWLASGTGKASVTKDVYH
TGTQAVQLEHAALSRTGLSQDLPITGGQTYKLSVWVKTKDVNSGIGVFVRTQLFKSIKGLDGKTQDDKIGSGPSTEELTG
TQDWMKREVIISTPEEAKYIRLEPFFETGTGTVWFDDISLVKWNSVTGLALDQTNVTLEKLQTAVLTAKVTPENVSGLSL
QWRVDNKDVVQVEAQGSLAVLTALKPGKAIVTVMTDDGAFAATSEITVKEASIPGDLTNDGKITVGDLAMAAAHFGKDTS
SPDWDTVKAADVNGDHKIDRSDLMWFTERIFP
>lcl|NZ_CP026520.1_prot_WP_042228612.1_2180 [locus_tag=PC41400_RS11185] [protein=polysaccharide lyase family 8 super-sandwich domain-containing protein] [protein_id=WP_042228612.1] [location=2437712..2442040] [gbkey=CDS]
MKKSILQVAVIALTIVALIFTSYPVTVQAQQPDPIDSMRDKWLAFLTGGQQIDLNDEDIRQAVEANAKKVTNDQKNGFWD
LLLKDPARKSLWADYASTTDSSHITLSYNRLKDMAIAYSTQGSNLYRNEQLKQDIIAGLDWMYTNRFNENKAKYNNWWDW
EIGSPLSLGDIMTLMYDHLSPEQVARYTRTIDKFIPDPTLRLGSTLKETGANLLDKALAVLIRGVIGKSETKIVQARSSI
SPVFPPVTSGDGFYSDGSFIQHNNIAYTGSYGSVLLGDMAKLLTILNDSPWPIQDPNLANTYDWITNSFEPLIYSGHMME
MVSGRAVSRYNNNTRGTAWTINRMSQFAPPDLAARYKSMVKEWVLSDTSVSNKYEGMPIRDLVLFKALMKDTSVVPRGDL
IVHHQFSAMDRVVHARKGYAYAISMSSSRISNYEGGINGEHTKGWYTGDGMTYLYNNDAAQFRDAFWPTVDSYRMPGTTS
DGLPRTGSKVTSKSWVGGASMDQLYGAAGMDLAPPGSKLTGKKSWFMFDNEIVALGAGITTSDSRKVETIVENRMINSAG
DNKLIVNGETMPSNLGWSKNLTGVKWANLQGTAAGADIGYYFPETANVSALREARSGSWRDINPTGPTDTITRNYASLAI
DHGTKPTDQSYSYVLLPTLNAAATQSYSENPDVRILSNTAKIQAVQETKLGLTGIHFWVPGTLDRIRAYQPVSLMMKEKG
DELTISVSDPTQSQSKVKVELAAAASQVLASDPSVKVLRTLPTLQLEVDTRGSLGKSHTIQLKINPDQEVELPGETDYEP
DPAAKVRIPVTEDTFVNGGTDANNNFGTRGFLNIRNGSGNTDRKVFLKFALPSSIQEIHSVSLNVYGETKDGSGNISDIG
VYEVPDDSWTETTLTFANKPAPGKQIDVATFTTPEQWRKFDVTPFVMTQQQGDKVASFALQQVGSNLASTIRSRKSEEGK
YQAYLEVLLKDTKAPTTSVRVDGNSGADQENYKDVTLHFTAVDDPNGWGVLRSEYRINGGKWMAVKAGQVIIREKGVQNI
EYRSIDKAGNMEEAQKLTVVLANADTELSGNQSVVKGEPLIVTYRIKKAVSDMYAHDLTLDYDAHVLEFVSAESMQRGVS
ILKSKAEDAGRLRLAIAGEGQEIQGNADLLKLTFKVKKATDAVSGKIEVVQAILADKQGSEMTVTPSSLHFSVQDSTEGP
GPVKPNPGDNGNGNGNGNGNGTGTDNGNSNGNGNGNGNGNGNGNGTGTDNGGHVQPEIPTDNTEAVVLNDLNGHWAEEKI
QEAVKKGLILGYPDGSFQPDKKITRAEFTSLLVRGLKLNEETELTFSDHDLKDHWAKSDIAKAVKKGIISGYEDNTFRAE
KDITRTELTMMAVKSLNLAEESPDLISFVDQADIPVWARGWVAAAVSKGLVTGRDNQTFAPLAPATRAEAVVILLNLLSL
RK
>lcl|NZ_CP026520.1_prot_WP_053228682.1_2181 [locus_tag=PC41400_RS11195] [protein=hypothetical protein] [protein_id=WP_053228682.1] [location=complement(2442792..2443304)] [gbkey=CDS]
MNSAAKYADVMLAPRLDFGPSQDPSLLLNYYFEYTLNFSKDVKRRIALEGCSPEDFFIASIQRPIVGRTEKEAEEMFQEL
QSLKPFYKIPKPLFFGSAEKVADQIQEWYEAGAMDILIVRQEHPSGLENFIELVVPILQDKGIFRTEYESNTLRGNLGLP
FPENRYAKRY
>lcl|NZ_CP026520.1_prot_WP_042228611.1_2182 [locus_tag=PC41400_RS11200] [protein=LLM class flavin-dependent oxidoreductase] [protein_id=WP_042228611.1] [location=complement(2443324..2444325)] [gbkey=CDS]
MRLSILDQSPISSNQTANDALIESMKLAQAGEALGYTRYWIAEHHGLPGLACSAPEVMLSYIGANTNRIRIGSGAVLLPH
YRPYKVAEVFNMLATLFPYRIDIGIGRAPGGSAEVTNALSDNFLQKVWELPALVKELFHFLEDDFSADHEYSKISASPIP
ENSPVPWLLGTSEKSALLAARNGMPYTFGHFMSDDDGIAIIQAYRDAFRPRKEAQAPQVIVTVSAICGETTEQAEEIAES
FLIWSLQGEGHQGVPSIKEAKQYKLTEKEKDSLNKMRQNMIIGNPHEVKRKLMELQTNYQADEIMINTITYSPKDRIQSY
RLIAKEIFSKENL
>lcl|NZ_CP026520.1_prot_WP_042228610.1_2183 [locus_tag=PC41400_RS11205] [protein=DUF3219 family protein] [protein_id=WP_042228610.1] [location=complement(2444332..2444628)] [gbkey=CDS]
MVKKMFLNGTLIYLRSYQEEKTNGLHKVSVDFNVTSEEYHDISTLLYSGTFDVEIPERDLTFRGTIYEYSTSLTNLYVKD
QVGQYKLTLMEVGHEAKG
>lcl|NZ_CP026520.1_prot_WP_042228609.1_2184 [locus_tag=PC41400_RS11210] [protein=LLM class oxidoreductase] [protein_id=WP_042228609.1] [location=complement(2444846..2445802)] [gbkey=CDS]
MDKFASHTGYSNMFKEKELTLGLYFPLESYEGSIAEMDLEHQMNLAKKAEEARFASLFVRDVPLNVPSFGDAGQLYDPWV
FLGYLAAHTKQISLGTGSIVTTLRHPLHVAKAAASVDRISGKRLVLGIATGDRPSEFSAFSVDREQRSNLFRESLDVMRK
VWNEEFPEIHSPRLNLRGGDLLPKPELKDIPILVTGHSTQTPKWIAENSDGWFYYPRNINNQAALIKEWRTLTNIFKPFS
QSLYIDLSENPNEDPTPIHLGFRTGHQFLIDYLHALKEIGVSHLAFNLKYGQRPVEEVIQELGEEVVPFFPALSNKEV
>lcl|NZ_CP026520.1_prot_WP_042228608.1_2185 [locus_tag=PC41400_RS11215] [protein=metalloregulator ArsR/SmtB family transcription factor] [protein_id=WP_042228608.1] [location=complement(2446140..2446451)] [gbkey=CDS]
MEQIEVFKALSNEARLQILEWLKNPAVYFSPQEGIDLEEIGVCVSHITEKLGMNQSTASQYLAILQRAGLITATRIGKWT
YYRRNEDAINQVGLFLKKELKKD
>lcl|NZ_CP026520.1_prot_WP_042228607.1_2186 [locus_tag=PC41400_RS11220] [protein=hypothetical protein] [protein_id=WP_042228607.1] [location=2446830..2447267] [gbkey=CDS]
MTGIDSIQPEVDEQAQDYISIEDEKNIRLALKYFNYTMLFIADEPENQQTQNFTLKQYVDFVKKKAAISNDDILDEKQIS
VFGHDAVQLTYKNHLDWNSRIDTLIVFKSDSQFYTIRNSSYTETYKSSLENFQQLMETMKFNEKQ
>lcl|NZ_CP026520.1_prot_WP_042228606.1_2187 [locus_tag=PC41400_RS11225] [protein=LD-carboxypeptidase] [protein_id=WP_042228606.1] [location=complement(2447340..2448263)] [gbkey=CDS]
MAIKPGALLPGDTVGIVTLGSPLSAETIQARIAYLQSLGLNVLLGRYVYSVNGYLAGTDRERAEDFMSVYADPRVKMILP
TRGGVGVADILPYLDYAFIRQNPKIVTGYSDITVLLNTLYKLADLITFHSLLLIDFRPDTPSYNFEQFFTATSSITAPRP
IANPPGIPLIGRVPGNVSGPIIGGNLTSLVGSLGTPFEVDTAGKILLLEETHEPVNTVYRYVDQLRLAGKLQACAGIVMG
ECTNCEAAYGKTYEDLISEVIVPLGKPLLTNLASAHGLYKAAVPIGANVNLNASAAALTLLEPAVRY
>lcl|NZ_CP026520.1_prot_WP_042228605.1_2188 [locus_tag=PC41400_RS11230] [protein=hypothetical protein] [protein_id=WP_042228605.1] [location=2448614..2449114] [gbkey=CDS]
MQSDTLCIIPPGKRKIWDRDATHGPASARDAYTGTLNRKARAYAEASGYDWIILSARYGFLMPADTVPGNYDTAFGAANA
GIISSDELARQWEEKGLHTYSRIEVLGGKKFRPIIGALAGTAAVYFPLENSRGIGDMLHSLDAWTVRMMKTDDEKGGRGM
KGPMND
>lcl|NZ_CP026520.1_prot_WP_042228604.1_2189 [locus_tag=PC41400_RS11235] [protein=UbiD family decarboxylase] [protein_id=WP_042228604.1] [location=complement(2449467..2451299)] [gbkey=CDS]
MYAHLEDCIADLEKNGHLIRIKEEVDPYLEMAAIHLKVYEAGGPALLFEKVKGSKFQAVSNLFGTLERSKFIFRHTWQDA
ENIIALRNDPVKALKQPFKTAGNGLAAFKALPLKKSGLPADFREISISDLPLIQHWPMDGGAFVTLPQVYSEDPEKPGIM
NSNLGMYRVQLTGNDYETDKEIGLHYQIHRGIGIHQDKANKQGAPLKVSIFVGGPPAHTLSAVMPLPEGLSEMTFAGLLA
GRRFRYSYAEGYCVSHDADFVITGEIHPGETKPEGPFGDHLGYYSLTHPFPVLRVHKVYARENAIWPFTVVGRPPQEDTS
FGELIHELTGDAIKQEIPGVKEVHAVDAAGVHPLLFAIGSERYTPYQQVKQPAEILTLANRVLGTGQLSLAKYLFMTAED
KQPLDTHKEKDFLAYILERIDLRRDIHFYTNTTIDTLDYSGTGLNTGSKVLFAAYGDKKRDLCTEVPTALTELSPFGEAQ
LVMPGVVAMRGPAFETYEKASLELEHLCGAIAGKGALPSCPMIVLCDDSEFLSADFANFLWATFTRSNPSHDIYGVNSAY
EYKHWACDNVIIDARVKPHQAPPLVPDPEVEKNIRRLFAPGASLGGVLKG
>lcl|NZ_CP026520.1_prot_WP_042228602.1_2190 [locus_tag=PC41400_RS11240] [protein=FixH family protein] [protein_id=WP_042228602.1] [location=2451617..2452066] [gbkey=CDS]
MKKILALTVLAASLALSACSQTASTSGGTKQESTDGLTVTIMAPEKVPLNQETVLKVEVKQGQEPVKDAGDIQFEIWNIK
NVDTAVNVVATYEDKGVYSVKNTFKEDGIYFVRTHVIARNQHIMPKQRIIAGNVSDEEIKTFEENVKKK
>lcl|NZ_CP026520.1_prot_WP_042228600.1_2191 [locus_tag=PC41400_RS11245] [protein=biotin transporter BioY] [protein_id=WP_042228600.1] [location=complement(2452140..2452739)] [gbkey=CDS]
MQLKRSSFTVRGITFSAMFAALLVALSFVNIQFGFSQVPFTLANFAIMLAGALLGARYGFFSIALVVGLTAIGLPLLHGR
GGLDLVLGPTGGFIWAYPFAALFIGLIVPRIKGSGVTAFVLTFVTLFVFGSLLLYVTGVPWMAHVVSNMTLTKALSVGCI
PYLPGDAVKAFIGALIVLPVRRIYPVSRLVGAESAWASR
>lcl|NZ_CP026520.1_prot_WP_053228681.1_2192 [locus_tag=PC41400_RS11250] [protein=ATP-binding cassette domain-containing protein] [protein_id=WP_053228681.1] [location=2452858..2453769] [gbkey=CDS]
MEYEHEKHMQSPYSMHEVSVVKRGEPEEGVSAADQTLLNEINMELNKGEWLAVVGRNGSGKSTLAKVMAGLIPAADGTIE
PGWTAGERMRMVLQNPDTQLIGETVYEDLCFGMENAGIDPERMLETARHVLGLVRLAGKRDDSCEALSGGQKQLTAVAGA
LAVGASVLLMDEVTSMLDSFSRSMVLGALSQLHEQGVTIVWITQRLDELALADKVIVVDEGKTVYSGDVRDFFYGSMSDD
DSRDKSRGNSAAASPCLELGYELPYTVEVTRELHRLGAFGLEERPVTPGELMEVLNRYGLPVR
>lcl|NZ_CP026520.1_prot_WP_042228597.1_2193 [locus_tag=PC41400_RS11255] [protein=ATP-binding cassette domain-containing protein] [protein_id=WP_042228597.1] [location=2453750..2455744] [gbkey=CDS]
MDYLYVKNLSKTVGGPDASFAVRDVSFTAVPGEITLLIGRSGAGKSTLLDMLGGLSVPDGGVIRYGDTGLWPGSKQSVRE
ARRKLALVFQHPHEQLFAATVRGEFHYTLRPFKLSGEEREARTAEALELFGLGPELLDERPHELSGGTQRRAALASVFAS
RPRWLLLDEPTAGLDHEAAARLTGTLRDWTRRTGGGALVATHDLDAFLPVADRIVMLSQGTVVHDCTPAELAARPAMLGD
AGLLLPEALALGSLLRWHGVDVPAGLPAPEETARAIAAALAAGARRGHGANADAGLSAPEERAAQAESAAASAHASVCAG
SASREAMLCGSVGASAASGSQAAAAGAGSAGPFECGRGVTPAIEDDSSAAETVQAGTSDRTPEEVRSPESGTQPASAASG
TSAPAAAPRPRALTVGRLFDPRTFWIGYVLLSAGILLQHSLIGLAVSAALTAAAVLLSQVPFRAVWKPLRAFLIFTGLSA
VIAGLTLGAGQGGSPVGFDPHAALSAVRALSNLMLVLVLGVLFPLTVGYLRMKQGVEQGLAFLGKLKFPVEAIALTASLM
FRFIPLIGGQWNRFGRIAKARGQGRSGKSGLGLAEFRIMALPFMFSLIRMADEFSTALEIRGYSRTGGRRTRSLTLRMKA
RDYAAMGVFLLVFLCLAALAGILG
>lcl|NZ_CP026520.1_prot_WP_042228596.1_2194 [locus_tag=PC41400_RS11260] [protein=hypothetical protein] [protein_id=WP_042228596.1] [location=complement(2455874..2456143)] [gbkey=CDS]
MAYDPQVVEATKVSDNPKNGLFEVVVVLKDRTHGRIFFERNLESGEIKASRISRLLKEPCPICRKDFLCNCLDRYLDNIS
GQALQKIES
>lcl|NZ_CP026520.1_prot_WP_042228594.1_2195 [locus_tag=PC41400_RS11265] [protein=hypothetical protein] [protein_id=WP_042228594.1] [location=2456252..2456527] [gbkey=CDS]
MEIDYAEVISAENELNLAVGVHFEDEPDSYYVVDILASPEGRIRSLELLFNGFSCKYTFKPDEKEQLVRYLNAHNPLALP
WVIPDEEAGVK
>lcl|NZ_CP026520.1_prot_WP_129112474.1_2196 [locus_tag=PC41400_RS11270] [protein=hypothetical protein] [protein_id=WP_129112474.1] [location=complement(2456470..2456724)] [gbkey=CDS]
MAISRARKLRDKQIREGKLDPAIKRRSWDGVQPVTKSTPTLQEKKRRVAHKHKRNSFPHTEDGSVYYFTPASSSGITQGR
ASGL
>lcl|NZ_CP026520.1_prot_WP_042228592.1_2197 [locus_tag=PC41400_RS11275] [protein=hypothetical protein] [protein_id=WP_042228592.1] [location=2456931..2457137] [gbkey=CDS]
MKIRIDLSVGGENIKEAFLQIEDRKVDHLTEEELLQAVEINIRSWADREIGISWEIVEFPVRPEEEEG
>lcl|NZ_CP026520.1_prot_WP_084706562.1_2198 [locus_tag=PC41400_RS11280] [protein=carboxymuconolactone decarboxylase family protein] [protein_id=WP_084706562.1] [location=2457283..2457648] [gbkey=CDS]
MSSVTHAGGGAMDAKIGAYKTGISQLQEQLPSVADAYHQFTGECFKDGALPGKTKQLIALGMGLFANNEVCTFYHVQEAL
SAGANAEEIRETVAVAAAIGGGHAMSQGVTRVEQALQGVVH
>lcl|NZ_CP026520.1_prot_WP_042228589.1_2199 [locus_tag=PC41400_RS11285] [protein=DUF2627 domain-containing protein] [protein_id=WP_042228589.1] [location=complement(2457896..2458156)] [gbkey=CDS]
MKDKLSRFVAIMLLVIPGVMATYGFLSMKDAFFHQFEAEGYIHWGKFVLGFILFGLGVAFIGGWTFFRDRKRNYVAPRFK
AKRRRQ
>lcl|NZ_CP026520.1_prot_WP_042228588.1_2200 [gene=lpdA] [locus_tag=PC41400_RS11290] [protein=dihydrolipoyl dehydrogenase] [protein_id=WP_042228588.1] [location=2458476..2459900] [gbkey=CDS]
MSGQAYDIVILGGGTGGYVAAIRAAQLGKKVALVEKDKLGGTCLHRGCIPSKALLRSAEVLATTRQAEEFGVIAPEVSLD
FTRVQSRKDGIVAQIHKGVQYLMKKNKIDVFNGIGRIMAPSIFSSRSGTLSVEMPDGSFETLIPEHLILATGSRPRVLPG
LEPDGEHILTSDEALALEALPDSLIIVGGGVIGVEWASMMNDFGVDVTVVEAGSRLVPTEDEEISRELDRLLRKRGVKVL
TNASVLADTLRKDASGVTVAAEQKGERVELAAGKILVSVGRAPNVEQFGLENTDVKIERGFIRVNGHMQTNIPHIYAIGD
VIGGLQLAHAASHEGVVAVEHICGLEPHTHEAHRIPRCVYTRPEIASVGWTEAEAKEKGFRIKTGKFGFKPLGKAIVYGE
NDGFIKVVIDADKNDLLGVHMIGPHVTDYIAEAALAQLLEASPWEIGSTIHPHPTLSEAFAEAVLAAEGRAIHI
>lcl|NZ_CP026520.1_prot_WP_042228586.1_2201 [locus_tag=PC41400_RS11295] [protein=thiamine pyrophosphate-dependent dehydrogenase E1 component subunit alpha] [protein_id=WP_042228586.1] [location=2460021..2461022] [gbkey=CDS]
MSIGQYVKHQQLGLSDERVIQMYTSMLLARKFDERSLLLQRSGKIAFHVSGVGQEAAQAGAAYALDAEKDYFLPYYRDYA
FVLSVGMTMKELMLSVFAKAEDPNSGGRQMPGHFGHSKHRIVTGSSPVTTQVPHAVGIALAAKMKGDELVSFVTFGEGSS
NQGDFHEGCNFAGVHKLPVILMCENNRYAISVPHHKQVAGNIADRAIGYGFPGVKVDGNDPLEVYRVVKEARERALRGEG
PTLIEADLYRVPPHSTSDNDLLYRTKEEVEENRAKDGLPKFRQYLIDCGIWSEEKDQELLTAIQQEINEATAYADAAPYP
TGEDTLKHVYAEA
>lcl|NZ_CP026520.1_prot_WP_042228583.1_2202 [locus_tag=PC41400_RS11300] [protein=alpha-ketoacid dehydrogenase subunit beta] [protein_id=WP_042228583.1] [location=2461049..2462029] [gbkey=CDS]
MAVITYLEAIRSAMQDEMQRDENVFVLGEDVGKGGVLNATKGLRDLYGESRVLDTPLAESAIVGTAIGAAMYGMKPIAEI
QFADFIFPATNQIISEAAKIRYRSNNDWSCPIVIRAPYGAAGGGAIYHSQCPESVFFGTPGLKMVAPSNPYDAKGLLLAA
IRDPDPVLYFEHKKCYLSLLAEVPEGDYIVPIGKAEVKREGTDLTVIAYGIAVHYALQAAEELAKEGISAHVLDLRTIQP
LDVEAILEAAAKTGKVLIAHEDNKTGGIGAEVAAIVAEELLYDLDAPIRRLCGPDVPAVGMNPPMEKFFLLNTDKLADAM
RELAQF
>lcl|NZ_CP026520.1_prot_WP_042228582.1_2203 [locus_tag=PC41400_RS11305] [protein=dihydrolipoamide acetyltransferase family protein] [protein_id=WP_042228582.1] [location=2462044..2463453] [gbkey=CDS]
MSESKFRLAEVPVPHLAESLVSATVGTWLKQPGDYIQQYDVLCELYTDKVNVEMPSPIEGRLVKILVGSGETAAVGDPIC
LLEVPESAEAASAGASGGPASQGDSAAAIAEESADQSMRGRYSPAVQRLAAENAVDLGRVKGTGFGGRITRKDVETYIAQ
GGAQAAPSGGSQAAAPLVQTPAVPQAHAPSAPAHQPSAEAPAANPYRPEPQIPVRSSGIHLSQNPPTPLEFDANSDSRGE
TFIDVTPIRNTIASRMRQSVSEIPHAWTMIEVDVTNLVTLRSKVKEEFMRREGINLTYLPFLLKAVVNAIKDYPIMNSVW
AVDKIIIKRDINLALAVGTEESVLTPVIKKADQMNIAGLALEADNLTKKARTGKLSLNDLQGGTFTVNNTGSFGSILSQP
IINYPQAAILTFESIVKRPVVIQDMIAVRSMVNICLSLDHRILDGVICGRFLQRVKENMEAYNLETKLY
>lcl|NZ_CP026520.1_prot_WP_016423989.1_2204 [gene=prli42] [locus_tag=PC41400_RS29735] [protein=stressosome-associated protein Prli42] [protein_id=WP_016423989.1] [location=complement(2463460..2463561)] [gbkey=CDS]
MKRNRLFKIVVYIMLFSMVASTLMVSLSFLTTS
>lcl|NZ_CP026520.1_prot_WP_042228580.1_2205 [locus_tag=PC41400_RS11310] [protein=M20/M25/M40 family metallo-hydrolase] [protein_id=WP_042228580.1] [location=2463633..2464769] [gbkey=CDS]
MIQQERLIKEFIELVQVDSVTQYEQEISVVLQTKFKELGLSVQTDDAAEKTGHGAGNLICMLEASGAPESTPCIFFTSHM
DTVAPGNGIKPQIGEDGIIRSDGTTILGSDDKAGIAAMFEAIRVLKEQNIAHGPIQFVITVGEESGLKGARAMDPSVMKA
EFGYALDSNGEIGEIAVAAPARAEIKIVMYGKSAHAGVNPEDGISAIQVASKAVSRMKLGRLDSETTANIGSFSGLGPLN
VVCDRVLLEAEARSIVQSKLEAQLAAMEQACKDTAADFGTTCDFEADIVYPAFMHDETSPVVKLAMKALEGMGCKTSTFH
SGGGSDANIFNGMGIPTVNLAVGYKEIHTTNEHIAVSDLVKTAEAVVAIVREVTEQAK
>lcl|NZ_CP026520.1_prot_WP_042228577.1_2206 [locus_tag=PC41400_RS11315] [protein=LCP family protein] [protein_id=WP_042228577.1] [location=2465028..2466053] [gbkey=CDS]
MKRERRRRRFWLWAVLPLALLLAVGGYYAWNAYHSWNMVYKERTETGVPQTTPVNPAPVRETLAKETYVFLMLGIDSGNV
EQGRSDTNMLAVADSSSQKITLLSIPRDTRIRIQGHGKEKLAHAYAYGGADLAVKTVEAFFGIPVDYYVAFNFNGVKEFV
DTLGGLSIDVEKNMDFDDRITHRHVYLKKGRQTLNGQQTLNYARYRGDADGDFGRMRRQQQVVRELLTQTVQYRNVAKIN
NLLKIMGGNVRMDIPARTIFQFASTFSALTGNDVVTMPLEATTGMIGGVSYMNITESERNRVAAELQRLIRKQPENTALP
LQSYGIYDLIDPGAGNRFGRP
>lcl|NZ_CP026520.1_prot_WP_084706561.1_2207 [gene=mciZ] [locus_tag=PC41400_RS11320] [protein=Z-ring formation inhibitor MciZ] [protein_id=WP_084706561.1] [location=complement(2466089..2466226)] [gbkey=CDS]
MKNYTASESLCLVGKAWEIKQFLQLAVQKGGRDLPLVVFLEQKSG
>lcl|NZ_CP026520.1_prot_WP_042228574.1_2208 [locus_tag=PC41400_RS11325] [protein=NUDIX hydrolase] [protein_id=WP_042228574.1] [location=2466443..2467009] [gbkey=CDS]
MSADQNNRKFEEVTIKTEPIFEGKIITLQVDHVRLPNGETATREIVKHPGAVAVLALTDDDRMIVVEQYRKALEKSQVEI
PAGKLDPGEEPIRAAVRELEEETGYSSESVRPVCSFYTSPGFADEILHLFVAENLVRGEARPDEDEFLECEAITLQEAKE
LIARGSISDAKTITAVYAWQAYKLTGSF
>lcl|NZ_CP026520.1_prot_WP_042228572.1_2209 [locus_tag=PC41400_RS11330] [protein=endonuclease Q family protein] [protein_id=WP_042228572.1] [location=2467009..2468250] [gbkey=CDS]
MLRTYYADLHIHIGRTEKGAAVKISGARNLTFYNIAREASDRKGIEIVGIIDCHSPAVQEEIEHYLAAGEMKELDGGGIR
YHQTTVLLGSEIEVRDPGMGPVHLLVFLPKLAQMKEFTEWMRPRMKNVELSSQRMYVTARELQAEVLRRGGIMIPAHIFT
PHKSIYGSGSTRMEHMLDLDGLAGVELGLSSDSEMAGYLPELDRYTYVTNSDAHSLGKIGREYNELRLAEPTFAELLLAL
RREGGRAVTANYGLNPRLGKYHRTYCLSCESILDESEAAVGRCLYCGSPKVVRGVMDRIMELAGREEPYVPDYRPPYYYQ
VPLEFIPGLGKRKLDGLLARFGTEMNILHRATREELAETVGDDIAGMILEARTGVLGLQVGGGGTYGKVKAPEKATSKAE
QADKGTKGTRAKR
>lcl|NZ_CP026520.1_prot_WP_042228896.1_2210 [locus_tag=PC41400_RS11335] [protein=PCYCGC motif-containing (lipo)protein] [protein_id=WP_042228896.1] [location=2468311..2468811] [gbkey=CDS]
MKKWGLLTATLLCASMLAAACESGGAKQEANGDGAAHHAANGDLQEKTASSAVLPKFLNKQREEIRLVYELAGKNADLLQ
SMPCYCGCGESAGHKSNQNCFIREMNADGSVVWDDHGTRCGICMEIAVKAISLKKDGKTPLEIRKQIDETYKGTAGKPTD
TPLPEA
>lcl|NZ_CP026520.1_prot_WP_042228570.1_2211 [gene=spoIIM] [locus_tag=PC41400_RS11340] [protein=stage II sporulation protein M] [protein_id=WP_042228570.1] [location=2468992..2469633] [gbkey=CDS]
MPAMQTLKQYFKEQFTLFLFVSVLFVVGVVFGAVMVNALSLEQKQELGRYLSSFYRTIQDGAQVDGKQSMLAAFSLHMKW
ILLIWTLGLSVIGLPLILVLDFLKGVLIGFSVGFMVGQLSWKGIAFALVSIAPPNLIVVPAILICSVVSISFSLFIIKNR
LVRRSGTIYHPFMRYSGVTLAMGVCMFGAALLEAFVSPTMMKWVTPMLAALTN
>lcl|NZ_CP026520.1_prot_WP_042228567.1_2212 [locus_tag=PC41400_RS11345] [protein=Fur family transcriptional regulator] [protein_id=WP_042228567.1] [location=2469736..2470257] [gbkey=CDS]
MEARIEKIKQQLQSQGYKLTPQREATVRVLLENEEDHLSAEDVFMLVKDKAPEIGLATVYRTLELLSDMHVLEKMNFGDG
VARYDLRSDSTHHHHHHLICVQCGAVDEIMEDWLGPLEERLAREYNFKVLDHRLDFQGICHRCLEKGADQPKNKDANPKH
HHHHTSAKPSTLV
>lcl|NZ_CP026520.1_prot_WP_042228564.1_2213 [locus_tag=PC41400_RS11350] [protein=SDR family NAD(P)-dependent oxidoreductase] [protein_id=WP_042228564.1] [location=2470495..2471304] [gbkey=CDS]
MEQTNTIHTGEYGLAGKTALITGASSGIGKAAALRLAKLGVKVCLVDIRKEYAEDVQEIIMRKGGEAIVAEADISDPEQV
KEAYAKLQETWGRLDIVLANAGINGKIAPIEDLEPEEWDATLQTNLKGTFLTVKYAVPLLKENGGSIVITSSINGTRIFK
NFGFTAYSSSKAGQAAFAKMAALELARYKIRVNVICPGSIDTNIEDSTERTPELEAIRIPVEYPETSMPLADRPGTADQV
ADLLVFLGSDLSSHVTGTEVYIDGAESLL
>lcl|NZ_CP026520.1_prot_WP_042228562.1_2214 [locus_tag=PC41400_RS11355] [protein=DUF4227 family protein] [protein_id=WP_042228562.1] [location=2471430..2471666] [gbkey=CDS]
MIVSVHKWLERTKFLILFVTLTYILYRVFALFSVWVEPVDKYRHPSGTSVKVFQQHAIQTEGSESMATRLRLFYWYGE
>lcl|NZ_CP026520.1_prot_WP_042228559.1_2215 [gene=xerD] [locus_tag=PC41400_RS11360] [protein=site-specific tyrosine recombinase XerD] [protein_id=WP_042228559.1] [location=2471749..2472648] [gbkey=CDS]
MREDLEVFMDYLEHEKGLSRNTLESYARDLQQYIEFLEQQNIRTLGETSKVHVVSYISRLKQLGRATATQSRTLVSIRSL
YGYMIRCGRMDFDPTRDIENPKQEKKLPQVLTMDEMERLLAAPQTDTSNGIRDKAMLEVLYATGIRVSELISLDVGDVNT
SLGFVRCVGSGAKERIIPLGSMAAKVVDEYVSASRPQLMKAHKPETALFVNHLGTRMTRQGFWKMLKRYAADAEVDKDIT
PHTLRHSFATHLLENGADLRSVQEMLGHADLSTTQVYTHVTKSKMKEVYDRTHPRAKMT
>lcl|NZ_CP026520.1_prot_WP_042228557.1_2216 [locus_tag=PC41400_RS11365] [protein=purine-nucleoside phosphorylase] [protein_id=WP_042228557.1] [location=2472900..2473721] [gbkey=CDS]
MSTEKQRVNEAAAYIEKKLGFRPTIGLILGSGLGVLADLIEEAVVIDYSDIPHFPVSTVEGHASELVAGTIKGRQVVMMK
GRFHQYEGYGAETVAFPVRVMKELGVQTLIVTNAAGGVNTSYEVGDLMVIQDHINFMFRNPLIGPNDDELGVRFPDMSEA
YSRRLGKLAHSVAEKQNFSLREGVYLGLLGPTYETPAEIRMIRTLGADAVGMSTVPEVIVARHAGIEVLGFSCISNMAAG
ILDQPLSHQEVMETTERVKPKFLQLVLGIIEEL
>lcl|NZ_CP026520.1_prot_WP_042228555.1_2217 [locus_tag=PC41400_RS11370] [protein=hypothetical protein] [protein_id=WP_042228555.1] [location=2473891..2474271] [gbkey=CDS]
MYKWIMFVLFIGASAMGVGFLFQDINARQAERAAEETAGASLKIAASNWKFDQAEYKIPKGQTKVSLLMKEGVHAIDITG
EGIDVKLDGTNKSQEVNFEKPGKYEIKCILPCGEGHNDMKSVLIVE
>lcl|NZ_CP026520.1_prot_WP_241688842.1_2218 [locus_tag=PC41400_RS11375] [protein=CHASE3 domain-containing protein] [protein_id=WP_241688842.1] [location=2474541..2477243] [gbkey=CDS]
MARAWRFSIRWKIMSGYLLILICLAASIFAVNGRIVSLQKEIDFVTEHDIQVHDLTSEVEKQLLDMETGQRGYVITGDDR
YLEPYEKAQANWLADYDTLHGLVADNPSQQKRLEELRSVIEKWIAEAGQPVIAWKKAGNTPQIQQYFQEDPGKKYMDRIR
VIIDTFRSTETQLTLDRVANLDERNKDMRSALYLMMLGVAVVAVATAILISRSIQGTIRKVIRAIDHLASSDGDLSERIE
VKSRDEIRDLGDATNRLLDNVQRETGEKTQVAELTTALQGFTDTAALSGMFVRKVAELFNAPYGVVYMLQTVEGEYGLHR
MAAYASDDGNGERSRMFKLGEGLVGQCAEENRMLAVDHLPENYVTVRSGLGEAAPRHILIAPASFKGSVTAVVEFASFEP
FTASQRQVLRELLTSFGIILNSVGSRMEVERLLQESRVMAEELQVQSEELRTQTEEMQVTNEELEAQYRVSEQKTLELEQ
ASLELQAKAEQVQLASQYKSEFLANMSHELRTPLNSMLLLSQFLAENQNNTLSEEELQYAELIHKAGNDLLALINDVLDL
SKVEAGKLEVNYAPFNLTELPDKMEHLFGKMLEDKDVAFKRSLEPGLPEIIQTDGQRLEQIVKNLLSNALKFTHYGEVTL
SLRRAKEDEENHLLPGLDWTTALAISVRDTGIGIPKDKQEHVFEAFRQADGATSRQYGGTGLGLSISRDLAGLLNGCIHL
ASEEGAGSTFTLYICCLDRADTAGMEAASALEAEASALPAPAKPKREASGRTGTVIAGKTVLIVDDDPRNVTALSAALES
RGMRVLTAGDGIECLALMDRTPNIDVVLMDIMMPGMDGYEAMTRIRERAEWADLPLIAVTAKAMKNDRDKCIEAGASDYI
SKPVDLNQLISLLNVWLNEN
>lcl|NZ_CP026520.1_prot_WP_272509712.1_2219 [locus_tag=PC41400_RS11380] [protein=D-alanyl-D-alanine carboxypeptidase family protein] [protein_id=WP_272509712.1] [location=2477438..2478697] [gbkey=CDS]
MWKKMSCAVLALSLLLSAAGPAVWAKEEKPQQVDLTPSAQSAVLMDADSGEIIVNKNMDTKLPPASITKIMTMLLIMEAV
ESGKIKMTDKVSTSEYAASMGGSQIFLEPGEEMSVEEMLKGIAMASGNDASVAMAEKLAGSEENFVAMMNERAKQLGMNN
THFSNCNGLPTENHYTSARDIAIMSRELLKHTGITKFTGSYQDHLRKDSAKPFWLVNTNKLVRFYSGADGLKTGYTSEAK
FCLSATAARDGFRVIAVVLGEPNTKTRNAEVTKLFDYAFAQYSNFPIFKAGDALGELKISKGDVASVPLTATRSYSILMK
KGESKEGIRHELQLNQDLKPPIAKGQPIGKIVVYKNNEVLTEYPIESPADVKKAGWWTLMKRSARSLFGGEQSRIFPLPK
SFWRFSSSFVSRQEMPKGL
>lcl|NZ_CP026520.1_prot_WP_042228554.1_2220 [gene=spoIIAA] [locus_tag=PC41400_RS11385] [protein=anti-sigma F factor antagonist] [protein_id=WP_042228554.1] [location=2478734..2479087] [gbkey=CDS]
MSLQIEFEHRSKALVVRLKGELDHHTADVVKTRMEDAIFRQDTKHLILSLKDLSFMDSSGLGVILGRYKQITGKGGRMVV
CDISPEVYRLFELSGMFKIITIEDNERSAISSLEVAL
>lcl|NZ_CP026520.1_prot_WP_042228552.1_2221 [gene=spoIIAB] [locus_tag=PC41400_RS11390] [protein=anti-sigma F factor] [protein_id=WP_042228552.1] [location=2479084..2479530] [gbkey=CDS]
MSESNFMKLQFASKSENEAFARVAVASFISQLDPTMNELTDIKTVISEAVTNCIIHGYDNKPDGVVTIMTEIVDDEVYIT
VEDNGAGIEDLEEAKQPLYTSKPELERSGMGFTIMENFMDEVDVVTEVGTGTRIRMKKRIESKKALVN
>lcl|NZ_CP026520.1_prot_WP_009674438.1_2222 [gene=sigF] [locus_tag=PC41400_RS11395] [protein=RNA polymerase sporulation sigma factor SigF] [protein_id=WP_009674438.1] [location=2479543..2480298] [gbkey=CDS]
MDVDLKGSSHSFLDDSEVKRLIALSQSGDSLARERLVNSNIRLVWSVVQRFLNRGYEAEDLFQIGCIGLLKSVDKFDLSY
DVKFSTYAVPMIIGEIQRFLRDDGTLKVSRSLKELASKIRKTKDELSKGMNRLPTIKEVADALGITPEEVVFAQEASKPL
SSIHETVFENDGDPITLMDQIADETQDKWFEKLALNEAIGALNERERLIVYLRYFRDQTQSEVASRLGISQVQVSRLEKK
ILQSIKDQIAQ
>lcl|NZ_CP026520.1_prot_WP_042228551.1_2223 [locus_tag=PC41400_RS11400] [protein=DUF378 domain-containing protein] [protein_id=WP_042228551.1] [location=complement(2480431..2480640)] [gbkey=CDS]
MSRIALTLIIVGALNWLLVGIFEWDLVSAIFGGEVHRTSSAFSRIIYTLVGLAGIYSLSFFFRDDASVR
>lcl|NZ_CP026520.1_prot_WP_164977119.1_2224 [locus_tag=PC41400_RS11405] [protein=stage V sporulation protein AA] [protein_id=WP_164977119.1] [location=2480833..2481471] [gbkey=CDS]
MNQTADACLYIRLRRRIRVPAREPVTLGRVAQMIVEPAYESTLRGLVLVQPKVKDGNRILIDMLSIVAAVKNVHPNLQIE
HFGEPHTLVEIDSGTRKPNLLFLAAVWVLLFIGSGLTIMNFHADVSMLEVHQRVYELVTGKKTEHPLFLQIPYSLGIGIG
MAVFFNHFFKKKFNEEPSPLEVEMFTYQENIDRYTIAEEYQSLNREKRESSP
>lcl|NZ_CP026520.1_prot_WP_042228548.1_2225 [locus_tag=PC41400_RS11410] [protein=stage V sporulation protein AB] [protein_id=WP_042228548.1] [location=2481468..2481887] [gbkey=CDS]
MITLLGWSAAVVTGLAGGIAVGSGMVAFLLVLDIIPRLLQVSRAQNRIRSCEIAVIAGSLVFTVMDFFNWTLSVPVWWTG
VFGLFAGAFIGMLSAALTEIINVLPVLAKRVGMTSYMVWLLMAMIAGKVLGSLFEWFIY
>lcl|NZ_CP026520.1_prot_WP_241688890.1_2226 [locus_tag=PC41400_RS11415] [protein=spore germination protein] [protein_id=WP_241688890.1] [location=2482440..2484161] [gbkey=CDS]
MEEDAFGIKKLLRDGNKGPIQASFSRKRSASDQGGGEQRANAEDHSAAADVRRSESTGQQKQASGGKGDKQGQKGKGPSN
GDESKPVDDPIPLSMQEVKDMLEERSGLGTSFDLVLREMVFGSIRVGFYYVNGLAKDQVLTMVLDRLSYADHAAMEQNPA
ESLKEYLIPHIQVSPYQTMKEVIGQVMAGATGVFIEGEKTALAIDAKNFPVRSTEEPDLERVVRGSRDGFVETLMTNVAL
VRRRVRDPKLRAEIMQVGKRTSTDICIGYIEDIANPKLVEAIRDKVKSVELDGIPLAEKQLEEAIVAKGWNPYPLVRYTE
RPDVAAAHLLEGHVILMSDTSPSVMIMPSTFFHHLQHAEEYRQLPIIGSYLRWVRFLGILASLFLLPLWYLMVLEPGLKP
TGLEFIGPQHPGQIPVLIQFLLAELGIDLMRMASVHTPTPLSVAMGLVAAILIGDIAVKTGLFTNEIIFYISIATIGMFA
TPSYELSLANRIVRLILLFSVALFKVPGLVIATTVWLMFLSLQRSHTAPYMWPFLPFNAMGLMNILFRLPVLKNKTRLSI
TRSKDGTRQPAPE
>lcl|NZ_CP026520.1_prot_WP_042228891.1_2227 [gene=lysA] [locus_tag=PC41400_RS11420] [protein=diaminopimelate decarboxylase] [protein_id=WP_042228891.1] [location=2484304..2485635] [gbkey=CDS]
MHLHGTSRVNALGHLEVGGVDTTELAARYGTPLYVVDEALVRRRCREFVDAFQASGLKFQVAYASKAFCVMAMCRIIAEE
GLSLDVVSDGELFTALQAGFPPERIHFHGNNKTPAEIEMALDAGIGCFVVDNFTELRMLDAIAGSKGKRVPILLRITPGV
EAHTHEYISTGQIDSKFGFDLGNGAAFEAVQESAAANHLLLLGVHSHIGSQIFEVEGFRVAAEKVAQFASQVRDELNVVF
SVINLGGGFGIRYVEGDTPLPMSTYVEAIAEAVKTRFEAVSYPMPEIWIEPGRSIVGDAGTTLYTVGTHKDIPGVRKYVS
VDGGMTDNPRPALYQSQYEAMLANRASEPSQETVSIAGKCCESGDMLIWDVELPLVQTGDILAVASTGAYNYSMASNYNR
IRRPAVVFVKDGMADMAVKRETYEDLVRNDVVPERMKQIQSVR
>lcl|NZ_CP026520.1_prot_WP_009674443.1_2228 [locus_tag=PC41400_RS11425] [protein=peptidylprolyl isomerase] [protein_id=WP_009674443.1] [location=2485807..2486238] [gbkey=CDS]
MAKQAKITMAGGGEVLIDLFEKDAPNTVANFEKLANSGFYNGLNFHRVIPGFVAQGGCPQGTGTGGPGYTIDCEINPNKH
ERGTLAMAHAGRNTGGSQFYICYQPQPHLDGQHTVFGKVTKGMEYVDAFKGRDTMEKVEIVEV
>lcl|NZ_CP026520.1_prot_WP_042228545.1_2229 [locus_tag=PC41400_RS11430] [protein=hypothetical protein] [protein_id=WP_042228545.1] [location=complement(2486287..2486550)] [gbkey=CDS]
MKRNSDFSRKGLKTAKNTAAVGLANLYPCKSDAIPVIVRGHVLDEGLAVSKWDKARLWDELDKAGVDSLLDVAYAEIRRD
ALILKSP
>lcl|NZ_CP026520.1_prot_WP_042228543.1_2230 [gene=ribD] [locus_tag=PC41400_RS11435] [protein=bifunctional diaminohydroxyphosphoribosylaminopyrimidine deaminase/5-amino-6-(5-phosphoribosylamino)uracil reductase RibD] [protein_id=WP_042228543.1] [location=2487661..2488800] [gbkey=CDS]
MLHPLQDETYMRLALQMAQAALGQTGINPVVGCVVVKDGRIVGMGAHLKRGEGHAEVLALQMAGREAEGATAYVTLEPCS
HYGKTPPCSDRLIESGIRRVVVASVDPNPLVAGSGIARLRSRGIDVEAGLLADEANALNEAFGKFIVTRLPFVTMKSAST
LDGKIAAKTGDSKWITGAPARELVHAMRHRHAAIMVGIGTVLADDPELTTRLSVPALHPLRIVVDSQLRTPLSARVLRQQ
DERPTLILTTDGADPAKREALEALGAEVLACGSGPHVDLPLAMRLLGEREVPSVLLEGGGRLSGAMLENRLVDKAVLFFA
PKWIGGGPEAPDNAAFAGFGRMSEAIRLARMTVETVGEDVCLTGYPQYAEKEASSCLPD
>lcl|NZ_CP026520.1_prot_WP_042228542.1_2231 [gene=ribE] [locus_tag=PC41400_RS11440] [protein=riboflavin synthase] [protein_id=WP_042228542.1] [location=2488785..2489453] [gbkey=CDS]
MFTGLIEEIGVMRRIHKQGQAMVLTLGAGLVLEDVKLGDSIAVNGVCLTVVAYDKETFSVDVMPETYRKTNLASLQTGAR
VNLERAMPANGRFGGHLVQGHVDGTARIRARTPEENAVVYRLEPDDDAILRYMIAGGSITIDGISLTLIDVTDSAFAVSI
IPHTLSQTILQDKQPGDSVNIECDVLGKYIEKLLRTGSGGRTPSRAGQGGLTEKFLAENGFF
>lcl|NZ_CP026520.1_prot_WP_042228540.1_2232 [locus_tag=PC41400_RS11445] [protein=bifunctional 3,4-dihydroxy-2-butanone-4-phosphate synthase/GTP cyclohydrolase II] [protein_id=WP_042228540.1] [location=2489493..2490716] [gbkey=CDS]
MTETETYSFHSIEEAVNDLAAGKIIIVVDDEDRENEGDFIALADKTTPEVINFMIKEGRGLVCAPITEERAQELDLPPMV
SRNTDYHGTAFTVSVDHAETTTGISAHERSLTIRALIDPEAKPLHFRRPGHIFPLIAKKGGVLRRAGHTEAAVDLARMSG
SYPAAVICEVIKEDGTMARVPDLMEVARTHGLKIITIKDLIHYRSEKENLVKREVEAKLPTDFGTFQAIAYTNQLDEKEH
VALVKGEIDASVPTLVRVHSECLTGDVFHSHRCDCGPQLAAALQQINDAGAGVLLYMRQEGRGIGLINKLKAYVLQEEGL
DTVEANVKLGFAPDLRDYGVGAQILKDLGIRQIRLLTNNPRKIKGLEGYGLEVVERVAIQMPHNEDNARYLHTKQEKLGH
LLQFDPN
>lcl|NZ_CP026520.1_prot_WP_009674421.1_2233 [gene=ribE] [locus_tag=PC41400_RS11450] [protein=6,7-dimethyl-8-ribityllumazine synthase] [protein_id=WP_009674421.1] [location=2490783..2491253] [gbkey=CDS]
MVKNYEGHLISQGQRYGIVVGRFNEFITSKLLGGALDALKRHGVEDTEVEVAWVPGAFEIPLIAQKLAETGKYDAVITLG
TVIRGSTPHFDYVCNEVAKGVSAISLKTGVPVIFGVLTTDSIEQAVERAGTKAGNKGWDAAVTAIEMANLTKQLQG
>lcl|NZ_CP026520.1_prot_WP_042228538.1_2234 [locus_tag=PC41400_RS11455] [protein=segregation/condensation protein A] [protein_id=WP_042228538.1] [location=2491351..2492133] [gbkey=CDS]
MKMLYKLEAFEGPLDLLLHLIDKSEVDIYNIPIKDITEQYMEYVQAMHELELEVTSEFLVMAATLLAIKSKMLLPKPPVL
DLDFDYYEEDDFDPRAELVQKLIEYRKYKAIADLLRDKEVERSLVFTREPEDLTPYVPQVQENPVKGLDVNDLMLAFQRT
LRKMVHRNQVTKIRRDEISVKDRITDLVELLRNKGGRLLFSKLFEKQCSREEIVVTFLAILELMKMKKIQCFQHRLFEDI
VIQAKEGELTDGLSTDEIDY
>lcl|NZ_CP026520.1_prot_PC41400_RS11460_2235 [gene=scpB] [locus_tag=PC41400_RS11460] [protein=SMC-Scp complex subunit ScpB] [pseudo=true] [partial=3'] [location=2492102..>2492704] [gbkey=CDS]
MDFQQMKSIIEGLLFVAGDEGLDKKQLAEILETDAHIVTELIEELQEDLRRGGRGIQITRVAGAYQMTTVPEHAPYFERL
AYSPSRSSLSQAALETLSIIAYKQPITRVEIEEIRGVKSERALHTLMAKELIVETGRAEAIGRPILYGTSKPFLEYFGLS
SIDDLPESPAFVGDFDLEEETKLLFDKLNGDETDQLTFEDL
>lcl|NZ_CP026520.1_prot_WP_042228534.1_2236 [locus_tag=PC41400_RS11465] [protein=DUF2953 domain-containing protein] [protein_id=WP_042228534.1] [location=2493077..2493811] [gbkey=CDS]
MLIGGIITAILLVLVLAAVLAMLSSVQLTLYWNRVGENDEMYVQVRALYGWIKYRFNVPAMQFNNLVDGLTLKINQTDQN
KMDITKKSKSRIDKAYIKAFYREGKDLLRHTQGLNAWFKQTMSAFRCTELKWSTGVGAGDAVQTAILAGTVWTLKSAVVG
LLTKYVPLSAVPKLAVNPQYNRTAFQIQFVCVIQAKAGALGLSGCRLIYRIFRTPGGMRAWFKLLRQVIPGDRTRAKRPE
PAST
>lcl|NZ_CP026520.1_prot_WP_042228524.1_2237 [gene=ytfJ] [locus_tag=PC41400_RS11470] [protein=GerW family sporulation protein] [protein_id=WP_042228524.1] [location=2493937..2494413] [gbkey=CDS]
MTDHPIQGLMKTAMENIKQMVDVNTIVGDPVETPDGSVIMPISKVGFGFAAGGSEFTTETLPEVGSRGGLEKAAVEMPFG
GGSGGGVSITPIAFLVVGKHGVRIVPLDNQTHLIERLIDSAPQIVDRVQSMIKQGGSASGLGGSGTSNINIEADTIKM
>lcl|NZ_CP026520.1_prot_WP_042228522.1_2238 [locus_tag=PC41400_RS11475] [protein=ArsB/NhaD family transporter] [protein_id=WP_042228522.1] [location=complement(2494505..2495812)] [gbkey=CDS]
MSQVWIALIVFLATYAVIISEKKNRAVVALLGAAIMVISGIVDVHKVYTEYIEWGTIALLVGMMILVGITNKTGVFQYVA
VRAAKLAKGDPMRILIFLSLLTATGSAMLDNVTTVLLVVPVTLSITRMLNINPVPFLISEIIASNIGGTATLIGDPPNIM
IGSANPHLTFNAFLSNLAPVTVLILIATLAMLRVIYRKQLKTSSEHKEKLMNLDEKAYIQDPVLMKKSLIILGLTIIGFV
LHSVIHVEAAVVAITGAVLLMLFGLKEKEVEEAFDSVEWVTIFFFAGLFALVGALQEVGIIKSLAQQVLNVTGGDVKLAA
PLILWGSGIASATIDNIPFVATMIPLIKDMAVNMQLPADSMEVNTLWWSLALGACLGGNGTLIGASANVIVAGIAAKEGK
GFSYWEFLKIGAPVTLVSLVISQLYLYFRYMIFLP
>lcl|NZ_CP026520.1_prot_WP_042228519.1_2239 [locus_tag=PC41400_RS11480] [protein=D-alanyl-D-alanine carboxypeptidase family protein] [protein_id=WP_042228519.1] [location=2496181..2497356] [gbkey=CDS]
MSKRIWVSGMLAFLCMLNVLGTAALAAEPPAIGTNAEAAALVDVKSGRVLYSKRGDRPMRIASLTKIMTAIVAIEESKLT
DIVKVSTRAAGKEGSSLYLKAGQEMSLKHMLYGLMLRSGNDAATAIAEHVGGSVEGFAVMMNRKAEQIGLDHSNFTNPSG
LDESADHYSSANDLAKLAAYSLKNPTFQEIVQTKMIKVPNPNESWEYTWFNKNKMLGLFDGADGVKTGYTKLAKRCLVSS
ATRNGQQLVVVTLNDGDDWADHTRLLRWGYEHYPLVPLLKKGQAVEGTPYVAEREFSYPLLSVEKDRVTSEPESFPLESL
DARLGEKGTLRLKLDDRLIGTVPLVDPASPRLKAADKAAFGFQATDADVTAWNAWSLSLRKAIRAMLSPGV
>lcl|NZ_CP026520.1_prot_WP_009674409.1_2240 [locus_tag=PC41400_RS11485] [protein=nucleoside recognition domain-containing protein] [protein_id=WP_009674409.1] [location=2497385..2497984] [gbkey=CDS]
MVNWIWLFFIVTGFAVAAVQGNVEAVTQAAFDGAKSGVTVCFGLISVLVFWLGMMRIAEDAGLLSKLAVLLRPVVRFLFP
SIPKDHPALGYIMSNMSANILGLGNAATPMGIKAMQELQKLNPHKDTASPAMCTLLALNTASITLIPTTLIAIRMNYNSM
NPSEIVGTTLLATIVSTAAAIIVDRWYRRRLPANPAVKG
>lcl|NZ_CP026520.1_prot_WP_042228518.1_2241 [locus_tag=PC41400_RS11490] [protein=spore maturation protein] [protein_id=WP_042228518.1] [location=2497992..2498528] [gbkey=CDS]
MYEWVNVLSAWAIPVLIVFIPLYAAYRKVPVYESFTDGAKEGFDTAIKIIPHLVGMMVAISVFRASGALELTLGWLKPLF
QALGIPPEVLPLAVLRPITGAGSLAFTTDLIAQFGPDSLVGRIASTVQGSTDTTLYVITVYFGAIGIRKAGYALKVGLIS
DVVGFIASIIICLLVFAT
>lcl|NZ_CP026520.1_prot_WP_042228515.1_2242 [locus_tag=PC41400_RS11495] [protein=pseudouridine synthase] [protein_id=WP_042228515.1] [location=2498831..2499577] [gbkey=CDS]
MERLQKVLAEAGIASRRKSEELILAGRVSVNEEVVTQLGVKVNPAEDLIMVDGRAIKQQTKVYVMLNKPKGVITSASDPK
GRKVVTDYLPGIKERVYPVGRLDYDTEGLLILTNDGEFANLMMHPRHHVAKTYHATVKGVVHGSVLDRLRAGIELEDGMT
APAEVEYVDVRPEQNESVISITIYEGRNRQVRRMFDAVSFPVAKLRRVQYGPLLLAGLPRAKYRVLRNDEINSLRDAALQ
SHKIQKGR
>lcl|NZ_CP026520.1_prot_WP_042228514.1_2243 [gene=resA] [locus_tag=PC41400_RS11500] [protein=thiol-disulfide oxidoreductase ResA] [protein_id=WP_042228514.1] [location=2499639..2500184] [gbkey=CDS]
MKKNKKWVQIIILGIVLIIGVFTIVNNLTSADDKKYPVEGSKAPDFTLTGLDNKSYKLSDFKGKGVLINFWGTFCPPCKK
EMPALQKQYDIWKDKNVEFLEVNLDTSRVTVQNFVDQYNLNLPILLDNEEKVRKQYGVVEYPTTFFVKPDGTVMKIHKGE
MDEAFIADTLKKLTDGTGGGQ
>lcl|NZ_CP026520.1_prot_WP_042228512.1_2244 [locus_tag=PC41400_RS11505] [protein=cytochrome c biogenesis protein ResB] [protein_id=WP_042228512.1] [location=2500221..2501888] [gbkey=CDS]
MFQNTKCDCGHQNHVGTLLCESCGKPLTEEAFSSEPLEMKYDGVARRSQRENPSWLDRVWNFFSSVKVAVTIIVLTLIGS
SLGTIYPQESNFIQVNPATYYKDTYGTLGHVYYLLGLSHTYESWWFKLLILMIGASLVICSLDRVLPLYRALSKQKIQKH
HSFLLRQKVAYDGRIPEGISEEEWTEQMARQLRKKHYRVHTDGAALLAEKYRFSRWGPYVLHIGLIIFLLAVLMRSFPGW
QMDEYMGLREGEIKKIPDTSLYIKNEKFTIDFYKPEEMTEEFKKRDRAVPKIYETEAVIYTCKADCDTPGKKPELQEVHR
QKIIVNEPLDYKGLLAFQYDYKQDPLLISVKPTLTNKQTGESYGSFELPMKDPQDTYQAGPYKLSLKGYFPEFGLDDKGE
PITKSNEAKAPAFIFQITGPGLSPNGEPYIYFPRQIDKQEFRQDEINGSIGAKFDLSVGSMENVTFSSFTSYLNIRTDRA
LPYIWIGAAIGMIGLIMGFYWQHRRVWLRIDDGRLVLGAHTNKNWFGVRKEVAASLNKTGITVDQKTLEKGGKAT
>lcl|NZ_CP026520.1_prot_WP_042228510.1_2245 [gene=ccsB] [locus_tag=PC41400_RS11510] [protein=c-type cytochrome biogenesis protein CcsB] [protein_id=WP_042228510.1] [location=2501885..2503141] [gbkey=CDS]
MNWLNFSGNALLVSFVIYCVAFLVFVLSIAGKTWSNRDPEVHGRKWGRAAYGLSVLGLVSQLLYFFTRWYGGGHIPTSNM
FEFMTMLGMMIMIAFIIVYAIYRTPVLGVFALPVVVLIIAYAAVFPNEVQPLIPALQSYWLKIHVTTAAAGEAFFAVGFA
AGLMYLIRTVNFQGKTASDRKEQRGVEFTLLIIFMLIAFIASVFTFNAAGYEAKFIKEQIVAKDGTQQKVSQTVTYVLPP
IVAPYQTTLESMDTFLGMDKPLFTAPSWMSGENAGRKFNTVIWTVLLGVLLYALLRLILRKPIGQAISPIMKGMDPEDLD
EIGYRSIAIGYPIFTLGALIFAMIWAHKAWGRFWGWDPKEVWALIVWLFYAVFLHLRLSRGWQGKRSAWLAVIGFIVVMF
TLVGVNLVIAGLHSYSGV
>lcl|NZ_CP026520.1_prot_WP_016423993.1_2246 [locus_tag=PC41400_RS11515] [protein=response regulator transcription factor] [protein_id=WP_016423993.1] [location=2503196..2503909] [gbkey=CDS]
MEQRLNILVVDDEERIRRLLRMYLEKEGYGIEEAEDGETALTLATERDFDLILLDLMLPGIDGTEVCARLRQVKSTPVIM
LTAKGEETNRVHGFEVGADDYVVKPFSPREVIYRVKAILRRSSATAFLSKDANSSNNIVFPHLVIEHDAHRVTAGGQEVA
LTPKEYELLHYLAVSPDKVFSREELLKDVWNYEFFGDLRTVDTHVKRLREKLNKVSPEAAAMITTVWGVGYKLEVPK
>lcl|NZ_CP026520.1_prot_WP_042228507.1_2247 [locus_tag=PC41400_RS11520] [protein=ATP-binding protein] [protein_id=WP_042228507.1] [location=2503911..2505779] [gbkey=CDS]
MFSLRSVVGKLWLTIIGLVAVILITLGVFLVQSLEQYFSQSEDQVHNLKQLAGDYSNGAALHLRDEEFYKMANQLLELQD
ARMIALDKDLKELNPSVEGKNQELLRFKTADFFSDTELKTVLNGQMLEKRIPYNKQGKINQRNDYIAVAVPLYHTPDGGA
KDTEGAIVLYQSLKSLEATQWDVFRLFVIVSIIGFLMTTVFAFFLFYRITRPLLQLKDAAGRISRGEYGTKVPILSSDEI
GELAKSFNVMGEDLQENVKALRHEKENLSSVLRSMTDAVMTFDAGGKVIVTNPQGDKIIQEWEELTWNPDGPPVNGGVPA
ALPLPLDRHIPAPLQPMFHQVVTEAREQTDKLHVENEVWSVAMTPLLSRDGVRGAVAVLRDVTEEFRLDKLRKDFVANVS
HELRTPLSMVQGYSEALLDDIAGSPEERKELAQVIYDEALRMGRLVRDLLDLARMEAGHLEMNIGRTDIGGLVKRMHRKF
TVLAREKDIRLNFVLPEGELVCTADEDRLEQVLTNLLDNAFRHTPGGKTITIKAEPCAYKSSPGCKLEVTDEGQGIPAED
VPYVFERFYKADKARTRGATGGTGLGLAIVKNIVDAHQGTITMRSKLGQGTTFTIVLPCKAC
>lcl|NZ_CP026520.1_prot_WP_042228505.1_2248 [gene=serA] [locus_tag=PC41400_RS11525] [protein=phosphoglycerate dehydrogenase] [protein_id=WP_042228505.1] [location=complement(2506084..2507667)] [gbkey=CDS]
MFKVLVSDPISDMGIQMLYDASDVEVDKKPGLSEDELVAIIGEYDALLVRSQTKVTPRIMEAAPKLKVIGRAGVGVDNID
LQAATKHGIVVINAPDGNTIATCEHTFAMMMSVARSIPQAYKKTVSGEWDRKSFVGVELRNKTLGIIGMGRIGSEVAKRA
KVFGMKVIGYDPFLTEDRAEKIGVKLGTVNDIAAAADFITVHTPLTNETRHIVGPAQFAIMKKGVRIINCARGGIIDENA
LVAAIENGTVAGAAFDVFEQEPPQASHPFLNNPSIIVTPHLGASTVEAQENVAIDVSEELLHILRNEPFKNAVNMPPVPA
TVMSKLQPYFNLGEKLGLIAGQLTEGPVNEIVVSYSGDLADVDTNPLTRYIVKGVFTQHLEGINLVNAMHMAKTRDINVT
TQQSSTTGTFTNLVTITVKSKQGDRTLAGTLLAGYGERIVRIDQYPVDFAPEGSLLLISHNDKPGIIGSVGTLLGSNDVN
IATMQVGRTIVGGSAIMVLTIDKTAPKEVIAELEKQPEIVNVRQLGI
>lcl|NZ_CP026520.1_prot_WP_042228504.1_2249 [locus_tag=PC41400_RS11530] [protein=rhodanese-like domain-containing protein] [protein_id=WP_042228504.1] [location=2508177..2508503] [gbkey=CDS]
MKHLAPEKFLELLDKPEMVKKIKIVDVREPYEWDYYHLDEPELIPMNTIPERLDELAGDVDLYVLCAHGVRSEMVCGYLE
RNGFDPDKLVNVQGGMAALAMLRGFAYD
>lcl|NZ_CP026520.1_prot_WP_042228502.1_2250 [locus_tag=PC41400_RS11535] [protein=CPBP family intramembrane metalloprotease] [protein_id=WP_042228502.1] [location=2508759..2509370] [gbkey=CDS]
MRKWNWKWKELKNIRIKTDIRVDDLNDRMLLINLYGTQAILLVIGVIMLFVQGADWRSLFSFAPGWSIVLWGGGIALAVL
AADLIVSKYVPEEVTDDGGVNKRIFGRRPLWHIALICLLVSICEELLFRAGLQHYLGPYWTSIVFAAIHIRYLQHWLMTG
LVFSISYGIGWIFIHTGSLWTAIFAHFMIDFISGCILRYGKEE
>lcl|NZ_CP026520.1_prot_WP_042228501.1_2251 [locus_tag=PC41400_RS11540] [protein=hypothetical protein] [protein_id=WP_042228501.1] [location=2509377..2509607] [gbkey=CDS]
MEGNKTQEIEYEQPILAESDLVHENEEESHYLPPRSSVHPSERAKVTRWFYRLLLFLFILLVMGLGWWGIRYTSGA
>lcl|NZ_CP026520.1_prot_WP_042228499.1_2252 [locus_tag=PC41400_RS11545] [protein=hypothetical protein] [protein_id=WP_042228499.1] [location=2509835..2510191] [gbkey=CDS]
MNLGSFLIGSVAGAALAMYAGKRIKNMRSAMFSSESSGIGSSLNDVFGQTKNFVSGSGHKESSKHQSHQSSSSSHGSGGS
HSSDGKEALENLIAKDSQVKAAVDEILVKNAASSSATH
>lcl|NZ_CP026520.1_prot_WP_042228496.1_2253 [locus_tag=PC41400_RS11550] [protein=genetic competence negative regulator] [protein_id=WP_042228496.1] [location=2510442..2511047] [gbkey=CDS]
MKIERLSPDKIRIFLTFDDLTERGIQKDDMWREIPKVHDLFSDMMDQAYSELGFEPAGPLAVEVFALPAQGMVVIVTRGS
MDLSLQADLLDDHDDEEVYEMEVTLEQSDIVSYAFRDFEDLLRASKVVNPMLTDGGALYTYKNKWILQLEPTELPEGKFH
TLIAVLSEYGDATNVTKAVLEEYGKAVIEANAVQELCRHFA
>lcl|NZ_CP026520.1_prot_WP_042228494.1_2254 [locus_tag=PC41400_RS11555] [protein=Glu/Leu/Phe/Val dehydrogenase] [protein_id=WP_042228494.1] [location=2511236..2512486] [gbkey=CDS]
MANTENLNVLESTQTIIEDALKKLGYSDEMYELLKEPLRVLTVRIPIRMDDGKVKVFTGYRAQHNDAVGPTKGGVRFHPG
VTEDEVKALSIWMSLKCGIVDLPYGGGKGGIICDPRSMSFRELEVLSRGYVRAISQLVGPNKDIPAPDVMTNSQIMAWML
DEYSRIREFDSPGFITGKPIVLGGSHGRETATAKGVTIMINKALEKRGIPLQGARVIVQGFGNAGSYLSKFMHEAGAKVV
GIADVNGALYNPEGLDIEYLLDRRDSFGTVTNLFPNVMPSTEILEQDCDILVPAAIENQITQENAHRIKAGIVVEAANGP
TTLEATKILTERGVLLVPDVLASAGGVVVSYFEWVQNNQGYYWTEEEVHSKLQEVMEKGFENVYNIHKTRGIDMRLAAYM
TGVRKMAEAARFRGWV
>lcl|NZ_CP026520.1_prot_WP_042228491.1_2255 [gene=prsW] [locus_tag=PC41400_RS11560] [protein=glutamic-type intramembrane protease PrsW] [protein_id=WP_042228491.1] [location=2512529..2513227] [gbkey=CDS]
MNVFSVLMAAVAPGISLLAYFYLKDRYETEPIHLVGKLFLCGVLLVIPAMVLQNSFMQEIGGDNHFTFAFFISGGLEEFL
KWFVVYHLIYRHDSFDEPYDGIVYAVAVSLGFATLENILYAFANGISMSSLLMRALLPVSGHALFGVMMGYYLGKAKFMR
SQERKYLLASLFLPILWHGAFDYILLVAANNWIWFMLPFMTLLWIRTLWKVDRANDRSPLRVVRGEEEFKIG
>lcl|NZ_CP026520.1_prot_WP_164977120.1_2256 [locus_tag=PC41400_RS29740] [protein=hypothetical protein] [protein_id=WP_164977120.1] [location=2513293..2513448] [gbkey=CDS]
MTVNRVKVTIRCRQCGERFILKGRRDKGKIETGFKRCICDNEHDLEVETED
>lcl|NZ_CP026520.1_prot_WP_042228487.1_2257 [gene=ypeB] [locus_tag=PC41400_RS11565] [protein=germination protein YpeB] [protein_id=WP_042228487.1] [location=2513583..2514968] [gbkey=CDS]
MYRKMSMVLFPVCAAALVGTGVWGYSQQKQKNQVSIKAENQYQRAFHDLSYHVEKLHTELGNTLAINSTSNDTYRKGLVN
VWRLTSQAQNEVNQLPLGLLPFNKTEEFLSNVSKFSYQTSTRDLTKKPLTDGELKTLSTLYQHSKEITGDLRGVQEKVIA
NNLRWMDVDLALGANKEPNDNTIIDGFKTVDKKVGSYNDLKEGPFTMSTQSKYTAQMLDGPEIDEQEVRKKAAAFLGVNN
PAIFHVVKNGNGAVEYESYTCTLPGKNGTDITMDFTKKGGHLIYFMNPRDVKSTKLQVEQTRAAAAQFLERHGYKNMTAV
SFDRHSNICNMTFAKQEKDTVIYPEKLVVNVALDDGQILGLQGTDFVFANRNHKNIDQEPRLSKEEARSHLSPQLKVTRH
TKALIVNDLNEEVLCHEFIGSLNGQTYRIYVNGEDGTEEKVETIRKEDAEQAKQQPVTQGT
>lcl|NZ_CP026520.1_prot_WP_042228486.1_2258 [locus_tag=PC41400_RS11570] [protein=flagellar brake domain-containing protein] [protein_id=WP_042228486.1] [location=2515098..2515757] [gbkey=CDS]
MAQLMKINQLLHLHVNTIDEEEAKQEYKARISDVLPDRLWIETPMNERTGRLKRLYTGDEVSAFFVTDGGVKNYFQTSVI
GFKEEGIRLVALAMPDPASITKIQRRSFLRIPAELEISVQAGETVPIVALTEDVGGGGVSFTCKDSEFPDNVTQVNCWLL
VTYKNGTLEHLPFKAEVVRVKTLETGRKIVMLQFSEITDRDRQKLIQYCFEKQFDFRKK
>lcl|NZ_CP026520.1_prot_WP_042228481.1_2259 [gene=cmk] [locus_tag=PC41400_RS11580] [protein=(d)CMP kinase] [protein_id=WP_042228481.1] [location=2516186..2516869] [gbkey=CDS]
MTKFNIAIDGPAGAGKSTIARKVAGELGFVYVDTGAMYRAVTWKVLQDGVNPLDTEAVARTAAGMKIELVPGPDGQLVFV
DGRDVSSDIRTKLVTDRVSQIAAIAEVRKLLVDKQKQMAASKGIVMDGRDIGTHVLPDAEVKVFLTASVRKRAERRFQEV
QGTAMATTLEEMEKDIARRDMLDEGREVSPLVQAADAVMLDSTDLTIPEVVESILEMCRMRTGGERA
>lcl|NZ_CP026520.1_prot_WP_042228478.1_2260 [locus_tag=PC41400_RS11585] [protein=lysophospholipid acyltransferase family protein] [protein_id=WP_042228478.1] [location=2516866..2517453] [gbkey=CDS]
MSFYHFARGVCNVVYSLLFRIRVIGKDNIPEHGGVLLCSNHISNLDPPLVGIRLKRKVHFMAKAELFQIPVLGLIISKVG
AFPVKRGGVSKETIKGTLNLLRDGNVLGIFPEGSRKNPGVGKKGAAMFALKSGAAVVPAAIIGDYKVFGRMTVIYGKPLD
LTPFSEAGSADLEQVTEYIMQEIQTLIREHRKEVA
>lcl|NZ_CP026520.1_prot_WP_009674424.1_2261 [gene=rpsA] [locus_tag=PC41400_RS11590] [protein=30S ribosomal protein S1] [protein_id=WP_009674424.1] [location=2517560..2518816] [gbkey=CDS]
MSEEIKNENQTNESQEAQEAMSQVAAVKKGDIVKGKIIKVEQDQAIVDIGYKYDGTIPLRELSSVQIENADQAVQEGQEI
ELKIVTIDDNKEKLVLSKRSVDSEKAWETLQQKLDEKVIIEAVVAEVVKGGLVVDVGVRGFVPASMVERSFVEDFSDYKG
RTLRLRVKEIDREKNKVILSQKDVLDEEYEAQKKDIISKIEVGKVYEGTVQRLTQFGAFVDIGGIDGLVHISEMAWHHVE
HPSEIVKEGDKVSVQVLKVDPEKDKVSLSLKATQEGPWEKAEKTFTTGEIITGTVKRLAGFGAFVEIAPGVEGLVHISQI
AHRHIATPQEVLKEGQEVKVKVLDVNAAEKRVSLSIKETEEAPAQTASSAPSNYSSRPERSSNHQKELAGNENVSLSNQG
LSMTLGERFGDKLSKFKK
>lcl|NZ_CP026520.1_prot_WP_042228477.1_2262 [gene=fni] [locus_tag=PC41400_RS11595] [protein=type 2 isopentenyl-diphosphate Delta-isomerase] [protein_id=WP_042228477.1] [location=2518844..2519899] [gbkey=CDS]
MRSSRKMDHVQFALQTGQSGNQGLTDVKLVPNCLPDASTESVALFTTIGELNLSSPIVVNAMTGGAHETEEINRELAVAA
RERGLAMAVGSQMSALRDPMQESSYRIVRQMNPDGLVFANLGSEATVEQASRAVDMLRADALQIHLNVMQELIMPEGDRS
FHGMLERVSRIAEGTGVPLIVKEVGFGIPLENARLLKSAGVQILDVGGSGGTNFAAIENVRRKLPLDWLNDWGMKTSVTL
IEALQVYERGRVMATGGIRTSEDIVKVLTLGASAAGMAGALLKILRNEGTQALIDHIDELHHGIVLMMTALGAVRIEQLW
TRPVIITGSTAEWCRARRIDLTEYACRTEAE
>lcl|NZ_CP026520.1_prot_WP_042228476.1_2263 [locus_tag=PC41400_RS11600] [protein=hypothetical protein] [protein_id=WP_042228476.1] [location=2519954..2520580] [gbkey=CDS]
MNPGYLASVLLAISVIFLLSGWREIFMRSISGRDILLFFMFALPAALMTFYVGGAQINGTVPVAFCTALLIGIRKKSLWE
GGHLFATGLLLGAVHLLMLELYSVDPVMIVLKPEWDIAIVLAVLVMGMHWGAAEQYVSLTTAMIAGDLAFACMHGGTMPV
HLGTPSFLDKWWLSFFTARLLTVFLESVYSGCRTTARQWIQRKRGWRK
>lcl|NZ_CP026520.1_prot_WP_042228474.1_2264 [locus_tag=PC41400_RS11605] [protein=YIEGIA family protein] [protein_id=WP_042228474.1] [location=2520577..2521485] [gbkey=CDS]
MIAHMLSQKETVAVFVGLFFGIASRLYLLRTDYRQYPTYPHGKIIHVSLGVIAAGLGSVAVPALLDKNYTAVTFLTVAAQ
QFRDVRNMERQSLMQVDQMELVPRGVSYIEGIAMVFEGRNYLVIFTSMMASLFTLLFAWYWGAVAGLAAVGVSIYFKTGK
NMTHIADIRISEVRLDGPNLYVGDIYIMNVGLESDRELIARQGVGLLAVPKNPDARVTLANLGQRQAILHDISTVLGVYR
DTGEPALMPMAKLHLDSGTLGIFVLPQVKDAQKAAAALKRVPVLESAIRMPTESAAGPHKEA
>lcl|NZ_CP026520.1_prot_WP_042228473.1_2265 [locus_tag=PC41400_RS11610] [protein=hypothetical protein] [protein_id=WP_042228473.1] [location=2521489..2521671] [gbkey=CDS]
MATIKAVAVVEGIKVDGGAPIFYAKDEAELQQLAFNLEKILDVNAHVLKPGTFILVDHKG
>lcl|NZ_CP026520.1_prot_WP_042228471.1_2266 [gene=der] [locus_tag=PC41400_RS11615] [protein=ribosome biogenesis GTPase Der] [protein_id=WP_042228471.1] [location=2521777..2523099] [gbkey=CDS]
MARPILAIVGRPNVGKSTIFNRIIGDRLAIVEDKPGVTRDRLYGIGEWLDREFSVIDTGGIEVEGEDQIMKSVRVQAELA
IEEADVIIFMVDAKAGVTSADQEVAQLLFRSRKPVVLAVNKVDNIQRQDEIYEFYSLGFGDPIGISGAHGIGIGDMLEAA
AAHFPDHADEEYGEEVIKVALIGRPNVGKSSLVNAILGEDRVIVSDQAGTTRDAIDTPFELDDQKYVIIDTAGMRKRGKV
YENTEKYSVMRAMKAIERADVVLVVINGEEGIIEQDKHIAGYAHEAGKAAIFVVNKWDVVEKDDKTMHQFTQKIRDHFLF
MTYAPVLYVSARTTQRVQKLLPMVNQVAEQHTMRVQTHVLNDVVSDAVMFNPPPTDKGKRLRINYATQVSVKPPTFVFFV
NEPDMMHFSYERYLENKIRGAFGFEGTPIRLFTRRKSDDE
>lcl|NZ_CP026520.1_prot_WP_042228470.1_2267 [gene=plsY] [locus_tag=PC41400_RS11620] [protein=glycerol-3-phosphate 1-O-acyltransferase PlsY] [protein_id=WP_042228470.1] [location=2523112..2523708] [gbkey=CDS]
MIYIIPAVAAYLLGSLSFSFIIGKLVKGIDIRSHGSGNAGATNTLRVLGKGPAVLVLVLDVVKAILAVWIGKWMAPDLIW
VHVACGLAVIIGHNWPVFFGFRGGKGVATAIGVTATLCFLPAIFAGIIAIASIFLTRYVSLGSLLFTAFLPFFVWGMGRP
VEILWGVIVIGVFTFWRHRTNISKLVQGRENKIGSKKA
>lcl|NZ_CP026520.1_prot_WP_042228468.1_2268 [locus_tag=PC41400_RS11625] [protein=NAD(P)H-dependent glycerol-3-phosphate dehydrogenase] [protein_id=WP_042228468.1] [location=2523744..2524784] [gbkey=CDS]
MTKHVSVLVAGSWGTALASVLAANDLDVTIWTRNAGQAEEINTLHTNSRYLQDIALSPRIRATTSIREAVENKDAVVFVV
PSAGMREVASQVREFLSPDTLVIHATKGFESLTLKRMTTVLAEELPQCDPGRIVVLSGPSHAEEVIRECPTTVVVASPTI
EAAEAAQDLFINSYFRVYTNPDVTGVEVGGALKNIIALGAGLSDGLRFGDNAKAALLTRGLAEIARLGAEMGANPMTFAG
LAGIGDLVVTCTSTHSRNWKAGYLLGQGEPLDQVLDKMGMVVEGVKTTKSAYALAQEYGVTMPITSELHHVLFEGKTPRA
AVEDLMGRGRTHEIEEVANQTDKWIL
>lcl|NZ_CP026520.1_prot_WP_042228466.1_2269 [locus_tag=PC41400_RS11630] [protein=stage VI sporulation protein F] [protein_id=WP_042228466.1] [location=2524873..2525145] [gbkey=CDS]
MPRNPKDILNTVKKKTGKSVTEKDIQKLASGVKPSTVQDEAQLRQLIKQVSQLVGIDVAEETVKEIVHAVKSSKLDGGNL
QQMMNLIMKR
>lcl|NZ_CP026520.1_prot_WP_042228465.1_2270 [locus_tag=PC41400_RS11635] [protein=2Fe-2S iron-sulfur cluster-binding protein] [protein_id=WP_042228465.1] [location=2525296..2525589] [gbkey=CDS]
MITLRTRKGDKTVESETGLTVLELALKHKLEWGHSCKRGTCARCRCFVKEGRERLAPPTEAELDRLEPEEIELGYRLGCQ
AVIQETGDVSVTHKPYF
>lcl|NZ_CP026520.1_prot_WP_042228463.1_2271 [locus_tag=PC41400_RS11640] [protein=hypothetical protein] [protein_id=WP_042228463.1] [location=2525600..2526235] [gbkey=CDS]
MKSERSKERRIALEMRPDLPLEEPLVRVMRQAARLLNGTGTGWVIGGSCALALQGVEIGAQPRDLDLYVDTDQAEAAARV
LEPYAVDEQKLDRTGIYESLLSHYAIGGIQVELVGGFKVCAAGSQYIVEAAYLREKHAEVGQLEGESLALMPLAHELLFN
ILRERPDRYEAIARTINGCPSRHEAALTDLLSRNSWGEPALSVVKRLIPGL
>lcl|NZ_CP026520.1_prot_WP_042228462.1_2272 [locus_tag=PC41400_RS11645] [protein=2Fe-2S iron-sulfur cluster-binding protein] [protein_id=WP_042228462.1] [location=2526355..2526708] [gbkey=CDS]
MSQTNAKVTFMPDNRSASVRLNTTLLDAARRANVHIRTRCSGQAACLMCKVHAGDQSGLSEPGKNERLKLGSQLQEGLRL
SCQARITGDVTVTVPEDPLKAAVRAQLAKQREEDDWP
>lcl|NZ_CP026520.1_prot_WP_241688843.1_2273 [locus_tag=PC41400_RS11650] [protein=hypothetical protein] [protein_id=WP_241688843.1] [location=2526754..2527509] [gbkey=CDS]
MTIAKGALAGRLTVLMAALFSLALLGGCMYPKEMRKENQTNPGEFIAVVQQSIDEYHTKKSVFPIKNSDMETPLYEKYVI
DFQKLKNSGYLSNLPGNAYESGGDFIYVLVDPDTKAKVKLFDLVAYRTTADIQKKVNDYISKNNGAVPGGETYSPGFKYV
DFGKIGMKTPEIKSVYNRQSLINFLVHDSGKVAIDYAPDLAQKIKEKQLQSALKDGQDLLPLLVEDSYFVPIMSFEYKWD
SATSKPVPFMK
>lcl|NZ_CP026520.1_prot_WP_042228459.1_2274 [gene=spoIVA] [locus_tag=PC41400_RS11655] [protein=stage IV sporulation protein A] [protein_id=WP_042228459.1] [location=2527800..2529278] [gbkey=CDS]
MEKVDIFKDIAERTGGDIYLGVVGAVRTGKSTFIKRFVESVVLPNIQSETDRIRATDELPQSAAGRTIMTTEPKFVPNNA
VQISVAEGLNVNVRLVDCVGYTVEGAKGYEDENGPRMISTPWFDEPIPFQEAAEIGTRKVIQEHSTLGVVVTTDGTISDI
PRSSYVLAEERVIAELKEVGKPFIVVINSQKPNSDPAQELRSELQSKYDIPVVTMSVATAGEDEMIGVLREVLYEFPVHE
VNVNLPSWVMVLPEKHWLRTHFENSVRDTVQDIRRLRDVERVVGQFSEYEFIDRAALAGMNMGQGVAEIDLYAPDELYDK
ILVEVVGVEIRGKDHLLQLMQEFTHAKREYDQFAEALEMVKTTGYGIAPPTLAEMALDEPELIRQGSRFGVRLKATAPSI
HMIRVDVESEFSPIIGTEKQSEELVRYLMQDFEDNPLKIWESDIFGRSLHSIVREGIQGKLAMMPDNARYKLQETLGRII
NEGSGGLIAIIL
>lcl|NZ_CP026520.1_prot_WP_009671985.1_2275 [locus_tag=PC41400_RS11660] [protein=HU family DNA-binding protein] [protein_id=WP_009671985.1] [location=2529559..2529831] [gbkey=CDS]
MNKSELIAKVSETSELSKKDATKAVDAVFDAISEALQQGDKVQLVGFGNFEVRERSARKGRNPQTGDEIEIPASKIPAFK
PGKALRDGIQ
>lcl|NZ_CP026520.1_prot_WP_042228457.1_2276 [locus_tag=PC41400_RS11670] [protein=PQQ-dependent sugar dehydrogenase] [protein_id=WP_042228457.1] [location=2530204..2531502] [gbkey=CDS]
MRQNGQTRVKSALLTAAAAAAIAVMASGCEGKPASTGGTGAGPAGTEAPGPANAGEARGSGSGNAPVVPELVEQNAADYT
REIVASGLNVPWELVFAADDRIFFTERPGQLRMIRDGKLMEKPLLKLQDPFISSGEGGLLGLALDPGFAVNSYLYVYHTY
AGEGETKNRVLRLKLQGDQALVDRVLLDGIPGAENHNGGRIKFGPDGMLYVTTGEKYDPPLAQEKNSLGGKILRIRPDGS
IPEDNPFQGSPVYSWGHRNAQGLAWDPRSGTLLSSEHGQSAHDELNVIVPGGNYGWPLIEGDESGSPKAAGALIPPLLHS
GKDTWAPSGMTFIRQGPWQGRLAVAGLRGTKLLLFTLEPERTSAPGAGASVPVYKLTKAEASFAGTFGRLRSVTEGPDGS
LYLLTNNRDGRGQPAGDDDKIIRLKPKRSGAE
>lcl|NZ_CP026520.1_prot_WP_009672001.1_2277 [gene=mtrB] [locus_tag=PC41400_RS11675] [protein=trp RNA-binding attenuation protein MtrB] [protein_id=WP_009672001.1] [location=2531589..2531810] [gbkey=CDS]
MDQTQGDYFIIKAKENGVHVIGLTRGHDTRFHHTEKLDKGEVMIAQFTEHTSAVKVRGKALIMTKFGNIETEQ
>lcl|NZ_CP026520.1_prot_WP_042228456.1_2278 [locus_tag=PC41400_RS11680] [protein=hypothetical protein] [protein_id=WP_042228456.1] [location=2531882..2532415] [gbkey=CDS]
MNKIARLLTIGGLSAAVVLTMAFVPFLDRKNGGQGSEAWRNISGPELTEANLPDYVNGLPLQLRVRKVELTHSILSLDLL
LPGTAEKKVVYGDLLRIAQSSLLRTSNISRVFIRIVEYGAKPGENGPQLVMAMEAARTEAKALSSLKGTDGEEKVAAELQ
RRFLLTFTKKWQRHFPE
>lcl|NZ_CP026520.1_prot_WP_042228455.1_2279 [locus_tag=PC41400_RS11685] [protein=heptaprenyl diphosphate synthase component 1] [protein_id=WP_042228455.1] [location=2532498..2533361] [gbkey=CDS]
MNPYRIPEIAKQYTDYDMIRQHTDLPDFPNFRAQLLYAFLSRDSRLNPSSELFALVTSLVQMGLDTHDEVTVSNEVKEKK
AARSRQLKVLAGDYFSSRFYHLLSQAGQIDLIKRLSTAICELNRLKTNLYVTMKHLKVTTEDYVRQSVDIKSHLFKSFGG
LMEEVNRRSWPEILEAFTRCEVIFKEIFRSESLQDFHGSWGYWHIIQNGSKEERKLLEAQEQDPAKLKALIHKYNVPSQL
YNLLVTQLQQLEAKVKQLDSDKLASELFHIGEPFFRFISKTRLLEER
>lcl|NZ_CP026520.1_prot_WP_042228890.1_2280 [locus_tag=PC41400_RS11690] [protein=demethylmenaquinone methyltransferase] [protein_id=WP_042228890.1] [location=2533376..2534125] [gbkey=CDS]
MQGKTKEQFVHSVFESIAPKYDKMNSILSFRRHKAWRKFTMAKMNVKPGSTAIDICCGTCDWTISMAKASGTGKMVGLDF
SQNMLDYGAGKVREEGLDNQIRLVHGNAMSLPFDDNTFDYATIGFALRNVPDVEQVLREMQRVVKPGGLVVSLELSKPTW
RPFKALYYFYFQKVLPLLGKLIAKSYEQYRWLPESLISFPDYKELAEIFRRTGLQKVEAYPLTFGIAALHIGVKGNSSQE
SKTDATTHS
>lcl|NZ_CP026520.1_prot_WP_042228454.1_2281 [locus_tag=PC41400_RS11695] [protein=UbiA-like polyprenyltransferase] [protein_id=WP_042228454.1] [location=2534106..2534990] [gbkey=CDS]
MRQLIHKTKIFLEMIKFEHSLFALPFAYMGSILGAVVETGDLPTWTQIMWITLAMIGARTAAMSLNRVIDKKYDKKNPRT
ANRAIPAGLISVAEVTVFIILSFLLLFWSAYHLNLLSVYLLPVAVFFLVFYSYTKRFTWACHFVLGITLGLAPLGGWVAI
TGGITPAALVLFITVAMWSAGFDLLYACQDVEFDREEGLHSYPSRFGIPVALITARVLHTLTAVGLISLFFMTNLSWWYL
VGIIIAYIILFYEHRLVSPTDLSKLNTAFFTMNGILSTVVFVFTLVDILVETFR
>lcl|NZ_CP026520.1_prot_WP_042228452.1_2282 [locus_tag=PC41400_RS11700] [protein=flavin prenyltransferase UbiX] [protein_id=WP_042228452.1] [location=2534987..2535622] [gbkey=CDS]
MKKRGWVVGITGASGAIYGVTLIKELLRRGVPVHLVITDAGWRVLHEELDWVAAKRKETLEAHFGGFPGTYTYYPLQDIG
AGIASGSFRTEGMVVMPCSMGTLAGISRGASDNLLERAADVMLKEGRRLILVPRETPLHAIHLENMLVLAKMGVKLIPAM
PAFYQRPETLDDIVAFLVGKVLDNMDLDHELFRRWGEGNDDNREETAHREN
>lcl|NZ_CP026520.1_prot_WP_042228450.1_2283 [locus_tag=PC41400_RS11705] [protein=menaquinone biosynthesis protein] [protein_id=WP_042228450.1] [location=2535582..2536445] [gbkey=CDS]
MTTTEKKLRIGRIEFTNAWPVFYHFPVQKLANSVEFVSQVPTGLNRALAAGEIDMSAISSFSYGENFRDYVLFPDLSVSA
YGHVNSLLLFHKKPLEQLKNGRFALPTTSATTTNLLKIIMNKKYGGNPTYSYAPPVLEDMMKDNDAALLIGDHAIKANWQ
DSGYTVTDLGQEWKEWTGHWMSFAVWAVRKDTIERSPELVKSIFDGFVESKRKSLEDLTGVINEAVLKIGGTESYWEHYF
RNLCYEFGPEQWSGLELYYRYAWELGFLPEEVSIQIWSDNTRVRVTE
>lcl|NZ_CP026520.1_prot_WP_042228449.1_2284 [locus_tag=PC41400_RS11710] [protein=polyprenyl synthetase family protein] [protein_id=WP_042228449.1] [location=2536442..2537416] [gbkey=CDS]
MKILNIYVRKKKDVGAIERELEANLQTDHALLTETSMHLLKAGGKRIRPVLVLLGGEFGTYDIERLKMIAVPLELFHSAS
LVHDDVIDDADTRRGQDTVKSKWDNRIAMYTGDFIFAKGLEIAARLEDPRVHMILSKALVEMSIGEMEQIRFFYHTEQTL
RDYLLRIKRKTALLIAISCQLGAMVAGADEYVSNRLYSFGYNVGLAFQIRDDILDLCGTEKEIGKPPGSDIKQGNITLPV
IFALQDEKLRGPILSEIERIQKVDGQTDVSRVLDMIRGSGGIEKSSALADRYIAKAIAALDTLPNIRAKQDLIQVAHFIG
NRTH
>lcl|NZ_CP026520.1_prot_WP_009671982.1_2285 [gene=ndk] [locus_tag=PC41400_RS11715] [protein=nucleoside-diphosphate kinase] [protein_id=WP_009671982.1] [location=2537541..2537984] [gbkey=CDS]
MEKTFLMVKPDGVQRGLIGEIVKRLEQKGFQLVAAKLMLMTREQAEEHYAEHKGKDFFEPLVEFITSGPVFAMVWQGEQV
IALSRILNGKTNVLEAAPGTIRGDFALHTRLNLIHGSDSPESAEREINHFFKPEEILDYKRDVSRWL
>lcl|NZ_CP026520.1_prot_WP_042228447.1_2286 [locus_tag=PC41400_RS11720] [protein=protein-glutamate O-methyltransferase CheR] [protein_id=WP_042228447.1] [location=2538104..2538883] [gbkey=CDS]
MENDDFPLFIRKIKDFTSIDLALYKEAQMKRRLTTLRIKKGYNTFSAYFDAMSKDKELFYEFLDRMTINVSEFWRNPNRW
EVMEQKFLPELRNASRNLKVWSAACSTGEEPYTLAMILSHMGALRDASILATDIDEGALEKARRGIYVERSLKDVPPHYV
KTYFKQEQLGFHVSDELKKAVKFQKQNLLEDRFDSGFDVIVCRNVIIYFTEEAKHRLYQKFSKALKPGGILFVGSTEQIF
SPAQYDLEPADTFFYRKKG
>lcl|NZ_CP026520.1_prot_WP_164977121.1_2287 [locus_tag=PC41400_RS29745] [protein=hypothetical protein] [protein_id=WP_164977121.1] [location=2538946..2539110] [gbkey=CDS]
MDKRKVISAYRRGFITMQECAQILGIDSVQILGMMNDPKMAGPSGPWQKQSAGG
>lcl|NZ_CP026520.1_prot_WP_042228446.1_2288 [gene=aroC] [locus_tag=PC41400_RS11725] [protein=chorismate synthase] [protein_id=WP_042228446.1] [location=2539306..2540469] [gbkey=CDS]
MRYLTAGETHGPQLTAIIEGFPSNVTIDFEAMNFQLHRRQKGHGRGRRMQIEKDTASIAGGVRHGRTTGAPIALVVENND
WKHWTSIMNIEEQPDTDEGKRRVNRPRPGHADLNGGLKYNQRDLRNILERSSARETTMRVACGAVARQLLEAFGIKLASH
VVRIGEVEAKSPQLPIDELIAITEESPVRVTDPEAEAAMIAAIDAAKADGDTLGGIVEVIVEGAPVGLGSHVQWDRKLDG
RIAQAVLSIQAFKGCEIGIGFEAAKLRGSSVHDEIQYDPEKGFRRASNRAGGFEGGMTTGEPIVVRGVMKPISTLYKPLQ
SVDIDTKEAFTAQVERSDTCAVPAAGVIMENVVAWEVAQAFMEKFGGDSLEEIRRNYENYLKQVESY
>lcl|NZ_CP026520.1_prot_WP_042228445.1_2289 [gene=aroB] [locus_tag=PC41400_RS11730] [protein=3-dehydroquinate synthase] [protein_id=WP_042228445.1] [location=2540471..2541565] [gbkey=CDS]
MMELKVDLGERSYPIYLGEGLLKDCAAYFDKHAISKRAPLLVVTDESVARHHLQPVMDSLTAGGYKASSFIVPPGEGSKS
LQMLEKIVTAALEAGLDRKSAIVALGGGVVGDLAGFAAASFMRGIAFVQMPTTILAHDSSVGGKVAVNHPLAKNIIGAFH
QPEMVLYDVTTLQTLPQRDISAGLSEMIKHGLIWDADFVTWLENNREKLLSLDPEALQYGLYQGCKVKSLVVSQDERENG
LRAILNLGHTIGHALEAVAGYNELLHGEAISIGMVGSARLAMKLGYSEDIYTVTKRILQSFGLPVRLPEQYDTARIMSAL
MHDKKFKEGTMTFIIPTAVGQVEIRNDVRREWVEEVVEQLKREA
>lcl|NZ_CP026520.1_prot_WP_042228443.1_2290 [gene=aroH] [locus_tag=PC41400_RS11735] [protein=chorismate mutase] [protein_id=WP_042228443.1] [location=2541568..2541945] [gbkey=CDS]
MFVRGIRGATTVESNEEGQILAATLELLEKIVRENEVKPDEISSVFITVTKELTATFPARAIRQMPGWDLVPLMCSIEID
VDNSLPKCIRLMVLANTTKRQDEMRHVYLKGAQVLRPDIVKLTNP
>lcl|NZ_CP026520.1_prot_WP_042228442.1_2291 [gene=trpE] [locus_tag=PC41400_RS11740] [protein=anthranilate synthase component I] [protein_id=WP_042228442.1] [location=2542296..2543840] [gbkey=CDS]
MNTELQEVIRLGKEYNLIPIVRCLLADTETPIRVFQHVAEEKRAFLLESVEGGIKWARYSFIGTDPFLIIRARDGKTHLE
TQGRTVTTEEKPLEALKSQLALFRSPSIEGLPRFTGGAVGFFGYDLLQYYENLPKHRVDDIQMDDIQFMFCDQVIAFDHF
KQQMNVIANVHIPDGASEETIAEAYRQTCQKLDATIQKLKKPLPSTMMTHSEVETTADLNGFRSNLSKEQYVSNVKAAQE
YIKAGDIFQVVLSQRFETDTDVSPLQVYRVLRTLNPSPYMYYLKMDDEIIVGTSPEMLVRVEDERVDTRAIAGTRPRGKT
PEHDLELEQDLLADQKERAEHLMLVDLVRNDVGRVAEYGTVSCDSFMQIERYSHVMHIVSNVSGTMNKEKDFFDAFLSCL
PAGTVSGAPKLRAMEIIAEFENEKRGTYSGAIGYLGFSGNMDSAITIRTIVFKNGKAYIQAGAGVVFDSDPEKEYEETVN
KARGMMTAIRTAEAMFRRAADNPNLDYYVHVPAL
>lcl|NZ_CP026520.1_prot_WP_042228440.1_2292 [gene=trpD] [locus_tag=PC41400_RS11745] [protein=anthranilate phosphoribosyltransferase] [protein_id=WP_042228440.1] [location=2543952..2544992] [gbkey=CDS]
MAANGTIQRAISTLIGGGNLSRDEAREVMLSVMDGEATPAQIGSVMTAMRIKGETIEEVTGFAEAMRLKATSVQTIQDNL
LDTCGTGGDGAGTFNISTASAIVAAAGGIRVAKHGNRAMSSSSGSADVLEALGVNIELDGNQAAQCLDQIGICFMYAQRY
HQSMKHAAGPRRELGVRTVFNLLGPLTNPAGADRQVLGVFERAKTELVAEVLGALGLKRAVVVSSYDGLDEISISAPTQV
TELKNGTVNTYAVTPDDLGLQTYSLQDIQGGNAAENAEIIRTIFTGATGAHREIVLANAGACFYVTGIAGTLQEGVKVAA
DVIDSGKAMDKLQQLAHYTGDLSYVS
>lcl|NZ_CP026520.1_prot_WP_042228439.1_2293 [gene=trpC] [locus_tag=PC41400_RS11750] [protein=indole-3-glycerol phosphate synthase TrpC] [protein_id=WP_042228439.1] [location=2544982..2545785] [gbkey=CDS]
MFLEKIVTTKRREVEELRGQVSSSELERRISGLQACRGFERALRDGRNRPTGLIAEVKKASPSKGLIRADFDPVRLASAY
AEAGADCISVLTDKDYFQGANDYLSAVREAVNVPLLRKDFTIDPLQVLEARVIGADAVLLIAAILKPSELKELKQLAQDV
GLDVLVEVHDRRELETVLEQDATLVGINNRNLKTFVTDIKTTEELAAIVPNDRILVSESGISSRDDLEFVRKAGAQAVLI
GEHFMRQPDVGQAVHDLMGERSMTRNF
>lcl|NZ_CP026520.1_prot_WP_042228437.1_2294 [locus_tag=PC41400_RS11755] [protein=phosphoribosylanthranilate isomerase] [protein_id=WP_042228437.1] [location=2545879..2546604] [gbkey=CDS]
MGSVSKRPDVKICGLQSVEVLKSIVHFPISQIGFVFAPSRRRITPEQASEMISFLHERAQQGHRVPKTVGVFVNPTRDEL
ADVLSVAKLDIIQFHGEEAPETCRSIKEQFGVELWKVSSIKNEEPPVNESSFSDIKDGVNGWIQRLDPYRDAVDAVLIDT
FDPVYGGGSGRTFSWDCIPAARKWALSAGMPLIVAGGLNPDNVGDLIEQYEIDGVDVSSGVETDGVKDIAKIAAFVERVT
G
>lcl|NZ_CP026520.1_prot_WP_174446793.1_2295 [gene=trpB] [locus_tag=PC41400_RS11760] [protein=tryptophan synthase subunit beta] [protein_id=WP_174446793.1] [location=2546604..2547809] [gbkey=CDS]
MAAVPDEHGRFGRFGGKYVPETLMNALIELEEAYRKYSQDPEFVKEVETLLHDYSGRPTRLYHAQRLSEKLGGAKIYLKR
EDLNHTGAHKINNTIGQALLAKRMGKKKVIAETGAGQHGVATATVAALLGLECKVFMGEEDMKRQQLNVFRMRLLGTEVV
PVMSGTRTLKDACNETLRYWVSNVHDTYYILGSVTGPHPYPMIVRDFQRVIGDETRSQILELEGRLPDAVIACVGGGSNA
MGMFYPFVGDEGVRLIATEAAGRGVDTEEHAATMTKGSPGVFQGSLSYVLQDEFGQVIPAHSISAGLDYPGVGPEHAYLK
DLKRAEYLPITDAEALDALQLLCKTEGIIPALESAHAVAQAIKTAPTMSPEELLIICLSGRGDKDVEAIMGYLGGADHES
H
>lcl|NZ_CP026520.1_prot_WP_042228434.1_2296 [gene=trpA] [locus_tag=PC41400_RS11765] [protein=tryptophan synthase subunit alpha] [protein_id=WP_042228434.1] [location=2547796..2548623] [gbkey=CDS]
MNRIDRKFAELRSAGKNALIPFITVGDPTADVTLDIIHELEKSGADLLELGVPYSDPLADGPVIQESSGRALKSGITVLK
VIEIAKQARSEGSELPMILFTYYNPLLRIGLERVFELMKESDISGVIIPDLPMEEDAEAKQLAERNGIHLIPLIAPTSNE
RVQKIVSRASGFIYCVSSMGVTGVRSEFHAGVDDFLASVKAATDLPIAIGFGISNREQVERFEKTCDAVVVGSAIVRRIG
EQEHRLQDESTKQDALKQIGEFVRDLKGASYQTQS
>lcl|NZ_CP026520.1_prot_WP_042228431.1_2297 [gene=hisC] [locus_tag=PC41400_RS11770] [protein=histidinol-phosphate transaminase] [protein_id=WP_042228431.1] [location=2548674..2549774] [gbkey=CDS]
MRPKQQVTKLPVYQPGKPIDEVKRELGLKHVIKLASNENPFGFSPKAKEAILADLDNLTLYPDGAGAEVTRAVAGHYGVN
ADQLIFGAGSDDVILMIARAYLVPGDETIMATHTFPQYKHNAEIEGAISVEVPLKDGKHDLDEMLRRVNERTKIVWICNP
NNPTGTIVTGQELEAFMNKVPSDVMVALDEAYYEYTTDEAYPDSLELLNRYPNIVILRTFSKIYGLASLRIGYGIGHPDV
IRTINQVREPFNTTRLAQAAAAAAIRDQDFVEQCRKLNREGIAYLLGEFRRLGLFSYPAHGNFILVDIGRPSGPVFDELL
RHGIIVRSGEKLDFPTMLRVTVGSREQNEAFVRALELVLHGAVQES
>lcl|NZ_CP026520.1_prot_WP_042228430.1_2298 [locus_tag=PC41400_RS11775] [protein=prephenate dehydrogenase] [protein_id=WP_042228430.1] [location=2549800..2550888] [gbkey=CDS]
MTTISIFGVGLIGGSLALCFKGKPDITVIGHSNNPTSVENMVKRRIVDEGTTSMQAAAEAADIIFLCVPVGALETYLERL
KQFSLKKGCLITDVGSTKLSVVQCAEKLRFKDAYFIGGHPMAGSERSGMEAASSHLFENAFYVLTPGTDTPVAEVERLAK
LLAYTQAQIIRVDAREHDDIVGAISHLPHMIAVALVNQIRTYNESNELYKNLAAGGFRDITRIASSDPFIWRDILINNRT
VLLKLLDDWDQQISVFRELLEKEDAEGIEKQFEVAGTFRKQVPDRRKGMLTPLYDIYADIPDHPGMIGQIATLLGKHRLN
LSNLQIIESRLDVPGVLRLSFRDQETADRASALLSEQFSIHV
>lcl|NZ_CP026520.1_prot_WP_042228429.1_2299 [locus_tag=PC41400_RS11780] [protein=sigma-70 family RNA polymerase sigma factor] [protein_id=WP_042228429.1] [location=2551330..2551926] [gbkey=CDS]
MTDTQLIREVKDGNVGKYNELMHRYERKIFVFIYHMLKSAQLELMAEDLCNETFYKAYRSLSSFREAEASFSTWLYTIAR
NTVLSEMRKHKIVKISLDNCGHTPQTSFESLPEQTVIRKEKVSMVRIAINSLPEKQRSALILREYDQLDYQEIASILGQT
VSSVKSLLFRARASVKLQMEPYYHTDNWLEECEGMTRR
>lcl|NZ_CP026520.1_prot_WP_042228427.1_2300 [locus_tag=PC41400_RS11785] [protein=zf-HC2 domain-containing protein] [protein_id=WP_042228427.1] [location=2551923..2552549] [gbkey=CDS]
MKCEDVQEWLLECIEGTASELRRQAVRVHSDSCKRCSEQFLAWQECRDLIRLSLRTDTPVPRVSIADSVMQRIGAYEPWR
YPAEAGAAASSGRAGKFRGAGRLAASFCLLLFVGVLLCLSLNSSPAFLGKGLSLTYEHAYAGSPGMEGASAMNGRTMKQA
VAGLNVSAVMPNKSTFVLPPYVHHLIALSLIGLFASFLFHAWIQRGKR
>lcl|NZ_CP026520.1_prot_WP_174446792.1_2301 [locus_tag=PC41400_RS11790] [protein=histidine phosphatase family protein] [protein_id=WP_174446792.1] [location=2552599..2553189] [gbkey=CDS]
MEEIILTTIGLIRHGSTEWNLLGKIQGALDTELTEEGREQARRLGTRLRGEAWDGIISSDLKRAKESAQLISERSGIPIA
GTDGRLREKGFGQVEGTTLQERLDRWGPDWLQLELGGETDEQVRSRWLSFYADLSVKYPRSRMLIVSHGAFIGRALETMR
LERRDDPLTNASLTIVRQGTAVWECTLYNCTEHLIV
>lcl|NZ_CP026520.1_prot_WP_042228425.1_2302 [locus_tag=PC41400_RS11795] [protein=hypothetical protein] [protein_id=WP_042228425.1] [location=2553363..2554478] [gbkey=CDS]
MNLADMLSYADIHDLNRIADTYACECNMHSKNELIQSILTTVSRRDIFEERVDALTMEDLRFLNTLLFEERGLLSLEELV
ARVQQAKFLKEEKENSNPRDTISKFKKHGWLFHGYSQQTKYLFQVPNDLKRRFGDVLTRNYKSRLIYTNDPAVYREEHTL
MADDVLQLMRYIRDHEVLLTHEDVMYKRQLSQILGGMAVKEELVGKTAWRFGYGRKFKEYPNRFSLIYDYCFFQELLQER
SGLLHLTEKGAEMTDAGIKPGQDELYRFWLRLYKGPVYNLQSIVQWTARLAGTWVTTASLAEVLCPLIRPFYYDTPESIF
EQRIIAMMMHLGLLAVGEEEHAGAVVRITPLGGKTIAGNMIAEEDKIELKE
>lcl|NZ_CP026520.1_prot_WP_042228423.1_2303 [locus_tag=PC41400_RS11800] [protein=gamma carbonic anhydrase family protein] [protein_id=WP_042228423.1] [location=2554582..2555097] [gbkey=CDS]
MLHPFKGLFPAIASSVYIAPGVQIIGDVSVGEDSTIWFNAVLRGDLAPIRIGNRCNIQDGCIGHVNTDQPLIVEDDVSVG
HGALIHGCRIGRGTLIGMGAIVLNGAEIGEYALVGAGSLVTEGTILPAYTLSLGTPAKVVRELTPQDLERMKRTTDNYVG
KGREYRMEPGK
>lcl|NZ_CP026520.1_prot_WP_042228421.1_2304 [locus_tag=PC41400_RS11805] [protein=IDEAL domain-containing protein] [protein_id=WP_042228421.1] [location=2555152..2555340] [gbkey=CDS]
MGRMNVSNEALLGLIAELVMDDAMRKYKENHLYQQIDSALAEGDEATFLALTAELKSIQSPS
>lcl|NZ_CP026520.1_prot_WP_042228420.1_2305 [locus_tag=PC41400_RS11810] [protein=DUF2487 family protein] [protein_id=WP_042228420.1] [location=2555548..2555997] [gbkey=CDS]
MKFSEITEKDWTDLQPYLDTCLLPVTGLGGGEQPWEAGDALESLRDVLDVLEIPYKGRVVTYPALHYCTPEDTDREYVER
TCGQLRKAGFTYVVVVTAHPEIGAWQLQNADLIVHISPEALRYPGQEAKKEIGERIEQLWFSKKSEEIR
>lcl|NZ_CP026520.1_prot_WP_016423998.1_2306 [locus_tag=PC41400_RS11815] [protein=ubiquinol-cytochrome c reductase iron-sulfur subunit] [protein_id=WP_016423998.1] [location=2556253..2556798] [gbkey=CDS]
MSERNNSNEHHGKKPVVRRQMSRRQFLSYTLGGTTAFMAAGMVAPMLRFAVDPVLQPKKASDWIKVVEEAKITNEPTSFT
FQVHQVDGWYESDVELTAWITKGQDGKIFALNPTCKHLGCTVNWNNNPEYKNQYFCPCHGAHYTADGKNLAVAPAPLDEY
DVKIDKGFVYLGQLHANTRAK
>lcl|NZ_CP026520.1_prot_WP_009675629.1_2307 [locus_tag=PC41400_RS11820] [protein=cytochrome b6] [protein_id=WP_009675629.1] [location=2556817..2557488] [gbkey=CDS]
MFKNMYNWIDERLDITPIWRDVADHEVPEHVNPAHHFSAFVYCFGGLTFFVTVIQILSGMFLTMYYTPDIINAYASVDYL
QHQVAFGVIVRGMHHWGASLVIVMMLLHTLRVFFTGSYKAPREMNWVVGMLIFFVMLGLGFTGYLLPWDNKAYYATQVGI
KIAASIPFLGQYVQTFLQGGDIVGAQTLTRFFALHVFFLPAALLALLAGHFLMIRRQGISGPL
>lcl|NZ_CP026520.1_prot_WP_042228418.1_2308 [locus_tag=PC41400_RS11825] [protein=menaquinol-cytochrome c reductase cytochrome b/c subunit] [protein_id=WP_042228418.1] [location=2557509..2558399] [gbkey=CDS]
MAHGQKDGEKVVYVGDSRVKKTVKNTVPPDYSSYPGKSEAFIPNFLLKEWMVAVVFLVGVLCLVISEPAPLGYPADPKNT
SFIPIPDWYFLFMYQLLKYPYTSDQYVVIGTLIVPGLMFGGLLLAPFLDTGKERRFYRRPIASSLALLTLIASAYLTYTS
WHHYQLELKATNTIPEHIKREEEMKEKKAAGGAASGGGGQQQKAAGVVAADDPAAEIYKKATCVSCHGGDLKGMPNNGIP
ALRGVGDKHDKEQIMNIIKNGQGGMQPQYKDNIAKGLTDADLDKLADWLAKQKKAQ
>lcl|NZ_CP026520.1_prot_WP_042228416.1_2309 [locus_tag=PC41400_RS11830] [protein=DUF1405 domain-containing protein] [protein_id=WP_042228416.1] [location=2558501..2559160] [gbkey=CDS]
MRNNATLHQALTLLRSRPMLWLLFWINALGTVYGYYWYKYQLIDTYDFEPAWYLPFVPDSPTASLFFTLTLLYLLRDERT
TGVGLGKRSLLRGFIEAFGLVTSFKYGIWAVAMIWSGYALGDPVAWQDWMLTLSHLGMAIEVLLFASVMTYSVSAVLLVA
VWAFANDWMDYTKGIFPRLSEVLMPHLDTIQWFTIGLSAAGILAALACLPYQKHRLRKV
>lcl|NZ_CP026520.1_prot_WP_042228414.1_2310 [locus_tag=PC41400_RS11835] [protein=sporulation protein YpjB] [protein_id=WP_042228414.1] [location=2559204..2560073] [gbkey=CDS]
MLFSSCKRIIGLLGAWAIFASAAVGCGPAPREAVPAASMTPVQMKTAESLNEAAEELYKLSEQGKTTEARIELEQLGELA
TKLTYDGSITIEGLHALTETITGAKKTYASVSFSQDEGRFAAAKIRLAADALTHRGQPMWLQYYKLIKDDLKRIETNVAA
RKKLEALLDYNKLYRHMSTIRPALLISRAPAEVEKLDSLLMFMRKGLAAPALDKTNLLQASGEFGRTIDGIFGRSDREAF
LPLPDTQEPVFWSLVVGAFIIGVLLFAGWRMFRLKPGYDTVRHKNREGF
>lcl|NZ_CP026520.1_prot_WP_042228413.1_2311 [locus_tag=PC41400_RS11840] [protein=YitT family protein] [protein_id=WP_042228413.1] [location=complement(2560070..2560948)] [gbkey=CDS]
MALALKDKTMPYLKQFLPIAFGTAIFSFGVHYFVIPNELMEGGLTGISLLLHYIFLLPPSATTLVLNVPLFYLGWRHFGS
KAMVYTIFGSLSSSFFLWVMEIMIGKGWIIPFATKQDYLLAALYAGFTIGLGLGIVFRFGGTTGGSDILAQLGNKWKGWS
VGQVILLIDLIVIGTSLLYIPKEKVLYSLVSVFIGSKMIDYITEGSHSAKAFTIITNEAENVSQAIQKELDRGVTLFPAK
GAFSRQEKEVVYCVVYRQEMRRLKELVKSIDPTAFIIINEVHEVLGEGFKVD
>lcl|NZ_CP026520.1_prot_WP_042228411.1_2312 [locus_tag=PC41400_RS11845] [protein=WGxxGxxG-CTERM domain-containing protein] [protein_id=WP_042228411.1] [location=complement(2561120..2561551)] [gbkey=CDS]
MKRLYSFLLCVMMFLTVSGTCIAQSAERGIKGEPGTLGYPDQKPSSYGTQQIKSGISGSTGTSDPSQYSNGGVGTSLPQP
PLNPVPKPGDYPPSKKYNVNLMPFDSKVSDSARRDDPTWDWLGFIGVLGLAGFIGKYRKKEFY
>lcl|NZ_CP026520.1_prot_WP_042228410.1_2313 [locus_tag=PC41400_RS11850] [protein=nucleotide pyrophosphohydrolase] [protein_id=WP_042228410.1] [location=2561726..2562064] [gbkey=CDS]
MPERSMKDLQQEVDTYISQFKEGYFSPLAMLARMSEEVGELAREVNHQFGEKPKKASEEENSIELELGDILFITICFANS
LGIDLTEAHDKVMKKFNTRDADRWTKIHTDSD
>lcl|NZ_CP026520.1_prot_WP_042228408.1_2314 [locus_tag=PC41400_RS11855] [protein=tetratricopeptide repeat protein] [protein_id=WP_042228408.1] [location=2562120..2562641] [gbkey=CDS]
MDGEQEIQKAYEAILNNDFEDAIAWFEKAVLSEPDNASYHYKLSITFARSGKLGKAIDHAVAAYELEPESEAYQYHLTTL
KAKVLIAEADKCLDQSREKEQLQTAIGLLKQAAALDPLSAEAYLKLAAAYAEADDYDLAVHAAKEAIRLEPLNKAAAGLL
DEYRRKLRQYLKP
>lcl|NZ_CP026520.1_prot_WP_042228406.1_2315 [gene=dapB] [locus_tag=PC41400_RS11860] [protein=4-hydroxy-tetrahydrodipicolinate reductase] [protein_id=WP_042228406.1] [location=2562675..2563481] [gbkey=CDS]
MTQEVINVAVAGASGRMGREVVKMVLGDSSLKLVAAVDRSSGPVDAASLVGLPDCGVKMSNDLELALVESKADVLVDFTG
PQSAVANTALAIRHGVRPVIGTTGFTTQDIEELDKQCKDKKIGGLIAPNFSIGAILMMKFASQAAKYFPHLEIIEYHGDQ
KLDAPSGTSVKTAELISQNREEFRQGNPKEEEQLEGARGAYYNGFRIHSVRLPGVFAQQEVIMGAYGESLKIRHDSYERA
AYMPGVNTAVHKVMTYEGMIYGFEHFID
>lcl|NZ_CP026520.1_prot_WP_241688891.1_2316 [locus_tag=PC41400_RS11865] [protein=methylglyoxal synthase] [protein_id=WP_241688891.1] [location=2563645..2564019] [gbkey=CDS]
MVNFVTAYEHVFDGNKLFATGTTGQRIMENTSLTIHRFMSGPLGGDQQIGAMVAQNEMDLIIFLRDPLMAQPHEPDIIAL
LRLSDVMGIPVATNIATAELLVRALDRGDFAWRELVHKYKPGEA
>lcl|NZ_CP026520.1_prot_WP_042228404.1_2317 [gene=bshB1] [locus_tag=PC41400_RS11870] [protein=bacillithiol biosynthesis deacetylase BshB1] [protein_id=WP_042228404.1] [location=2564019..2564735] [gbkey=CDS]
METVDILVFGAHADDAEIGMAGTILKHTKAGYKVGLCDLTRSEMSSNGTPENRLEEANKAAEAMGIVLRSNLGLPDRGLS
AGSKDQIDAITEEIRLRKPKIVFAPYWEDRHPDHIACSYLVQEAVFNAKLRRYLPETEAFQVESLFFYFINDTIDPDIVV
DVSDAYPGKKEVLQAYRSQFTPPQVGEDIVLTPLNRNYVERVEARDALLGQKRMMAYAEGFMSKLPVALQYFLPKKGT
>lcl|NZ_CP026520.1_prot_WP_042228403.1_2318 [gene=bshA] [locus_tag=PC41400_RS11875] [protein=N-acetyl-alpha-D-glucosaminyl L-malate synthase BshA] [protein_id=WP_042228403.1] [location=2564738..2565916] [gbkey=CDS]
MNDKLKIGITCYPTLGGSGVVATELGKLLAEKGHEVHFITHSMPFRLGKFHRNIYYHEVEVSDYYVFRYPPYDLSLASKL
AQVAQMQKLDLLHVHYAIPHAVCALLAKQMVGPDLKVVTTLHGTDITVLAQDQSISDLIRYAINESDAVTAVSNDLIAET
RETLDISRDIDLTYNFVDKRVYYPRDVKSLRKEFARDDEKILLHISNFRPVKRVTDVVDIFKRVSEKIPSRLLFVGEGPE
LSKVISYVKELGLSDRVNFCGKQDDVAQLISLGDVMLLPSEKESFGLVALEAMACGVPTVGSIAGGIPELVTHGETGFLA
EIGDTEKMADYAVELLSDPQQYEKMSAACLNRARTTFCNDWITMQYEEIYYRVLGKELPPSHRVDKQACAER
>lcl|NZ_CP026520.1_prot_WP_042228401.1_2319 [locus_tag=PC41400_RS11880] [protein=CCA tRNA nucleotidyltransferase] [protein_id=WP_042228401.1] [location=2565951..2567585] [gbkey=CDS]
MEEKNRTMIEEGRRVLEKLQSRQHEAYFVGGFVRDFVLERPVKDVDIATSAKPEEVMEAFSKTVPTGLQHGTVTVLTGAY
SFEVTTFRKETEYEGFRRPAEVEYITDLEEDLKRRDFTMNAMAMDVEGNLIDPFGGRSDMDRGVLRCVGEAAERFGEDAL
RMLRCIRFSAEYGLEIEEATWQALLRQAPLLRHIAMERVRAELQRMVGGADPARATGLLLASGLWRHAKMPLALHFAAWG
HPHAQAVIAHLSSIPAEEDRWAVLLLLIGSQQLEAVRAELRALTFPGEDIARLTRILAVHAHLQSGDAEDAAVRPRPEAQ
ASAEERAAARFKLAVLAHGKEAVRGWLRAAGLLRAAWRQRQAGTGAASPAGTGRTGQYGTLPGKSAQSGLAVAQGESDCL
EASRAGAQRPEQDAQPSAKHGQPSQVVPRLASGRYEASGGLSTDEEMHTSADKACLEELFSFMTEEANVYLENGEPWIRV
MPVYSVKDLAVGGSDLMAQSGQKAGPWIGETLNRLLERVALGQLENILEILMTEAFRPTEMRDE
>lcl|NZ_CP026520.1_prot_WP_042228400.1_2320 [locus_tag=PC41400_RS11885] [protein=biotin--[acetyl-CoA-carboxylase] ligase] [protein_id=WP_042228400.1] [location=2567589..2568563] [gbkey=CDS]
MTVRERIVDLFRSNPGVYLSGEQISRELGCSRTAVWKHIESLRQDGFQFEAVTRKGYRLTEEPDKLRKAHIEEALQTRFM
GRHIEVREEVESTMNEAHRLVAEGAEEGTLIFAEEQTSGRGRMGRHWHSPKGKGIWMSLILKPRVPVYFVPQLTLLTAVA
LCRAIRKVCGIDIGIKWPNDLLVKGRKVSGILLEMSGEDERLKYVIIGVGITANLTEADFPEDLKPVATSLALESGAPVR
REAMIAAFLNELEQLYDLYLENGFAPIRIMWEAMSVSLNCPIRVKTSQETVEGRADSLDDSGALTVITDDGRKVKIYSGD
VELR
>lcl|NZ_CP026520.1_prot_WP_042228399.1_2321 [gene=panB] [locus_tag=PC41400_RS11890] [protein=3-methyl-2-oxobutanoate hydroxymethyltransferase] [protein_id=WP_042228399.1] [location=2568861..2569721] [gbkey=CDS]
MAQERKPVTTVKLQKMKASKEPISMVTAYDYLTAKLADEAGVDMILIGDSLGNVIQGHESTLPVTLDEMIYHTKAVTRAV
KSAFVVTDLPFMTYHGSLDTTLQNAGRIMREGLSKGVKLEGGAEIAHAVKALTQAGVPVVGHLGLTPQSVHVLGGYRIQG
KNSEQARKLMEDAKALEEAGAIAIVLELVTEELAEWITSKLSIPTIGIGAGASCDGQVLVFHDMLGYEPDAPAKKFVKAY
AQVGLAIREGIGQYVAEVKGRKFPAEQHGFKGDNEVLQYLYGTAVK
>lcl|NZ_CP026520.1_prot_WP_042228398.1_2322 [gene=panC] [locus_tag=PC41400_RS11895] [protein=pantoate--beta-alanine ligase] [protein_id=WP_042228398.1] [location=2569761..2570642] [gbkey=CDS]
MEVIQSVALLRSTLQQKKSGIGASLKAPVGFVPTMGYLHEGHLSLIKAAREASEIVVLSIFVNPLQFGPTEDLDRYPRDT
ERDLKLAEEAGVDIVFLPQVEEMYPGPVKTTVSVSGITSVLCGASRPVHFDGVTTVVAKLLNIVQPDKAFFGLKDAQQVA
VVSQMVQDLNMNVDIVACPILREEDGLAMSSRNVFLSKDEREQALVLSRSLKAAEAWVREHASFRAEDLRGAVKAVIEEA
PLADIDYVEVLTYPQLEQPEAVEAGVPVRLIVALAVKFGATRLIDNTILTLGK
>lcl|NZ_CP026520.1_prot_WP_009675514.1_2323 [gene=panD] [locus_tag=PC41400_RS11900] [protein=aspartate 1-decarboxylase] [protein_id=WP_009675514.1] [location=2570647..2571030] [gbkey=CDS]
MFRTMMKAKIHRATVTEANLNYIGSITIDEDLLDQVDMLPNEKVQIVNNNNGARLETYIIPGPRGTGVICLNGAAARLVQ
TGDTVIIISYAMMTEEEARRHKPTVAIMGENNKVAQLLKEEMHATVL
>lcl|NZ_CP026520.1_prot_WP_042228396.1_2324 [locus_tag=PC41400_RS11905] [protein=tetratricopeptide repeat protein] [protein_id=WP_042228396.1] [location=2571122..2571760] [gbkey=CDS]
MFKHLFATMNEVLDEILLQLPNAKGALKAELEEELGALKAISDDCIEHWLLFEEKLGKTLSVGNASVTSSLASAADSNQL
QNVPQPAEEAQPHSAEGKDAVEFIRGQGFYMLLMYEQATREFEALLTREPDFLLARAYLAMGYLRQGDNEEAARHFTLLL
ALTENKQMKAISYNAMGCIQFQNRNLEQALEYFKKAYHADPDCLELFSASNE
>lcl|NZ_CP026520.1_prot_WP_042228395.1_2325 [gene=dinG] [locus_tag=PC41400_RS11910] [protein=ATP-dependent DNA helicase DinG] [protein_id=WP_042228395.1] [location=2571879..2574752] [gbkey=CDS]
MKFAVLDFETTGTGPADDIIQVGLVIIDDMEVTDRYTSLVNPGISIPSSITELTGITDEMVEHAPKVDDIMAEMYPLLFD
CVLVAHHASFDLGFLQRALRKTGYPDFTGRVLDTIDMLRIVFPSLPTYQLGMVASAFGIEHERPHQADSDADVTAEIWTM
SLDRLLSLPLLTVQRLAQILPAETSDLGWFMSEIARFKEQQTPLDQSIHQYYRQFALNVQDWGDEDPVRDENDVLELSDS
FEDVYDQVKEALKAKFEHFESREAQEIMIREIEEAFQNDRHLMIEAGTGTGKSLGYLIPSLYYGIKHDEKVVVSTHTINL
QEQLRERDIPLLHDIFPVNFKASVLKGRSHYLCLRKFENRISMQEFDTGKEDAITAAQMVVWLSQTQNGDEEELHFGNKG
AAFWHTVSSDAESCQNRHCPWFKKCFYHRARHDANNSDVVITNHSLLFTDMKAENRVLPAYKHLVLDEAHHFEEVASKHL
GKELHYLGLTNQLLWLFKDSRNGQLPALRTKLQRYDDDYRAATWMNGIDNLFPKTLLIKEAWDLLCEKFYELLQTNSEPG
QIDPGSLVMRLKKESLPPNWSDFTDLEEILEQHIQEVYKSVDSLVQDLKEVQEEYDVQSFVTDLGGVAKELQAAKETLRF
FMKMADDNFVYWVEAGTYNKSRSINLISVPMDVSPLIREHFFDNKESVILTSATLSVNKSFQYSADQLGLKLPVEEEENP
KLKTVQLPSPFQYRQQALVCIPRDFPMIRGRSGEEEFLKALTESLTDVAVATRGRMLVLFTSNRMLKLAHAVLKENLRPH
GIHVLGQGVDTGNRSKLTRLFQEHKACVLLGTSSFWEGVDIPGEALTCLAIVRLPFQPPNHPLVEAKSENLKKRNQNPFM
KLSVPQAVIRFKQGFGRLVRRASDKGIVLIYDTRVIETQYGKHFLYSLPGPKIEHMPVERIVPRIQEWMGENSLEVT
>lcl|NZ_CP026520.1_prot_WP_042228394.1_2326 [locus_tag=PC41400_RS11915] [protein=redox-sensing transcriptional repressor Rex] [protein_id=WP_042228394.1] [location=2574739..2575362] [gbkey=CDS]
MKSLKISEAVVKRLPIYLRFLNELSKKNVLTVSSQDLGQKLNLNPAQIRKDLAYFGEFGKKGIGYDVTYLIEKIRQILKL
DRQIPVALVGSGNLGRALCNYNIYLRDQMKITAIFDSSPDKIGQVINNMSVLPMEHLADVIRDSSIRIGIITVPAPEAQN
VANQFVKAGIEAILNFAPSILKVPEEIRVHHADFTAELQSLAYYLPE
>lcl|NZ_CP026520.1_prot_WP_042228393.1_2327 [locus_tag=PC41400_RS11920] [protein=inorganic phosphate transporter] [protein_id=WP_042228393.1] [location=complement(2575463..2576476)] [gbkey=CDS]
MTTTVLITLIAIVFLALAFDFINGFHDTANAVATSISTRAMSPKVAIISAACLNFVGALYSSEVAKTIGSGIADPSKINH
GEFIVIAALLSAIIWNLATWYYAIPSSSSHTLIGSIVGAVLAGAGADYVKWGGLGKIVLILVLSPIVAFVAGYIIMTIIK
YLVILTGNKSRSKLNRVFKFFQILSAWLLSFSHGGNDAQKAMGIIVFGLVAAGAQTTMDVPLWVKIAAATAMGLGTSIGG
WKIIKTISKNLIKIEPINGFASDLTSSIVIQTSTHMGMPLSTTHVISSAIIGSGSVLRFHEVKWGTVTRMIVTWIITIPI
SIILSYVLYTVLFKFFV
>lcl|NZ_CP026520.1_prot_WP_009675522.1_2328 [locus_tag=PC41400_RS11925] [protein=DUF47 family protein] [protein_id=WP_009675522.1] [location=complement(2576500..2577129)] [gbkey=CDS]
MASLFSSKRGDVDFLNLLIRAAENTLHASQMFRNAMMGDKIPENFFQMIKDIESKGDQITHEIFKELNKVFITPLDREDI
MELASKLDDVLDGIEATAARFDYLNISQTDKVMQEFSAVIVSSCQHMLDAFKLLSKKKYMQIPQHTVEINALENEADRLM
REGIREIFITKRDPYEDFKLKELYERLEQTTDACEDVANILESVVLRYS
>lcl|NZ_CP026520.1_prot_WP_042228392.1_2329 [locus_tag=PC41400_RS11930] [protein=amidohydrolase] [protein_id=WP_042228392.1] [location=2577371..2578669] [gbkey=CDS]
MKKTIIINGLFVTGDAAAPVIEGCMVMEGNQLTYIGKDAPQPLETYDEVIRGDRKLYMPGLVNTHGHAAMTLLRGYGDDL
ALQVWLEEKMWPMEGKFTAQDVRYGTLLAILEMIKGGTTTFVDMYDHMNEVAQAVSDSGMRARLTRGVIGLCPADVQEAK
LREAIAFAKDWNGAANGRITTMMAPHAPYTCPPDYIERFVQAAHDLDLPIHTHMSETLKEVEANAEQYGARPVEHLEKLG
VFTRPTLVAHGVHLTDEEIAVLAKYNVGVSHNPGSNLKLASGVARVPELLKAGINVSLGTDGAASNNNLDMFEEMRLAAL
IHKGVTGDPTAVPAAAALSMGTSMGARSIWLDDIGTLQPGMKADFIALDIDQPHFLPRTDYVSHIVYSASAHDVVDVCVD
GSWIVSGRECLTLDEEKIKREFESCFARLTNS
>lcl|NZ_CP026520.1_prot_WP_042228391.1_2330 [locus_tag=PC41400_RS11935] [protein=DUF5590 domain-containing protein] [protein_id=WP_042228391.1] [location=2578698..2579201] [gbkey=CDS]
MLKKAIWIGLFVVLTSGFGINRFYVNVQANHWERNSEAVQAAYEKTTMTKANRVEAFHGARSYEVVYGEDKLGNGLIVWV
PLKQDGTGLAIDEIHTEMTAENITEEAVRGIMAKKNAKLDILRVNAGEFKGKYVWEVFYKDEQDRAFYDYYNLKTGALED
TFKLSLQ
>lcl|NZ_CP026520.1_prot_WP_042228389.1_2331 [locus_tag=PC41400_RS11940] [protein=hypothetical protein] [protein_id=WP_042228389.1] [location=2579333..2579866] [gbkey=CDS]
MRKIRLLPVLLSVVITAFFLFGGWFAYQTYAIGKPLGNIVKETPGVEQSQVTYNDKEVSIRLSLKPDTSLREVYGRITKE
GASTIGSRGVKLSVNTKESPDLDKWWAQALFDVAQAMETRHYADIPTALEKHKTDKDSNLVVKTEMDEQNVYVHLQKGDQ
HKYVVLPRNAEKVGVFK
>lcl|NZ_CP026520.1_prot_WP_042228388.1_2332 [locus_tag=PC41400_RS11945] [protein=AAA family ATPase] [protein_id=WP_042228388.1] [location=2579863..2581383] [gbkey=CDS]
MSKYGKEIAIGAAPVLFIFLAFIKVNIIPLLFTAALITAVLYMMKSRGGSGIGIGAGAGTRGKSRQPSYLSFDQIGGQER
AKKELVEALDFLVKQEETKKLGIRPLKGILLTGPPGTGKTLMAKAAAHHTNSVYIAASGSEFVEMYVGVGASRIRTLFQE
ARTKAQKENKENAVIFIDEIDVIGGKRDGGQQREYDQTLNQLLTEMDGIHTNTTPRILIIAATNRKEMLDPALLRPGRFD
RHIDVELPDKKGRLSILNIHAQNKPLDEAADLEKIAEQTFGFSGAQLESVLNEAAIYALRGQQETVMQKDLLAAIDKVMM
GEQTDREATQEEKIRVAYHELGHAIIAELVRPDSVAQVSLRPRGKALGYVRHNPPQDQYLYSKEAIEDQIMIALAGAAAE
EIIYGGRSTGSSNDFEKSIDMVRTMISSGLTSLGIVDLKMVPQQKLMEVNNAILDELMARTKELLTKHRIVFEQSLDILL
KEEVISGQQFRDLLEGSKRHLEPVLA
>lcl|NZ_CP026520.1_prot_WP_042228387.1_2333 [locus_tag=PC41400_RS11950] [protein=3-hydroxyacyl-CoA dehydrogenase NAD-binding domain-containing protein] [protein_id=WP_042228387.1] [location=2581768..2582640] [gbkey=CDS]
MQYKKIGVIGGGTMGQGIAEMLAAKGLEVVLVEKDNERMEHSWAMIELSLDKQLEKWAITGAEKKLILSRIHKTTDLKEL
PSCHMVIETISEDLDLKKSLFRKLDEICGPDLILASNTSTLSLTELGASTKHPERVIGLHFIHPVAKIDLVEIIRGLKTS
EDTYMLTRRFVEDVIQKTAIKVYESPGFVTSRLICLLINEAVHTLAERVASAEDIDLAMRKGYEFRHGPLEMADRFGLDS
VLAALDRMFREFGELKYRPSFLLKQMVHAGHLGVKTGEGFFRYDKEGDRM
>lcl|NZ_CP026520.1_prot_WP_042228386.1_2334 [locus_tag=PC41400_RS11955] [protein=acetate kinase] [protein_id=WP_042228386.1] [location=2582637..2583848] [gbkey=CDS]
MNILVINAGSSSVKYQLYNMKDETVLAKGRVERIGLDSAILTHEPEGKTEVREVSEILEHTTAIKKVLDKLVHKEHGVLK
TVNEIDAVGHRIVHGGEAFSSSVVVTEEVKKEIKRLFDLAPLHNPAHMLGINAVEANMGDVPQVVVFDTAFHQTMPSHAY
MYAIPKIWYSKHKVRRYGFHGTSHQYVSERAARFLDRPLEELKIVSAHIGNGASCTAIQGGKSVDTSMGLTPLEGLMMGT
RSGDLDPAVVTFAMGKEDLSLNEVNSMLNKHSGLQAISGLSSDMREITEAMDSGDKNAKLAFDMYTYRLRKFIGSYAAAM
NGIDVLIFTAGVGENSVILRKAVCEGLSYLGLGLDSERNINGWGIEKRISTDDSKVEVLVIPTNEEWMIARDTFELVNGL
NEA
>lcl|NZ_CP026520.1_prot_WP_042228384.1_2335 [gene=asnS] [locus_tag=PC41400_RS11960] [protein=asparagine--tRNA ligase] [protein_id=WP_042228384.1] [location=2583953..2585248] [gbkey=CDS]
MSSVSTIHEVKDHVGESVRIGCWLHNKRKSGKIQFLQLRDGTGFIQAVVVKSEVSEEVWEAASKLTQESSLYVTGSVREE
PRSKSGYELTVEGVEIIQITQDYPITPKEHGVDFLMDHRHLWLRAPRQRAILVIRAEIIRAIQQFFDDKGFQLVDPPILT
PSSCEGTTNLFHTKYFDEDAYLTQSGQLYMEAAAMALGKVYSFGPTFRAEKSKTRRHLIEFWMIEPEMAFVDHEESLRVQ
EQFVSHVVQTVLKNCRSELETLGRDISKLEKIQAPFPRITYDEAVQFLQDNGHEFPWGDDFGAPHETAIAEKHEKPVFIT
HYPTGIKAFYMKPDPNRPEVVLCADMIAPEGYGEIIGGSQRIDDPELMEERFKEHELSSEAYQWYMDLRKYGTVPHSGFG
LGLERTVAWICGLEHVRETIPFPRLLYRLYP
>lcl|NZ_CP026520.1_prot_WP_042228383.1_2336 [locus_tag=PC41400_RS11965] [protein=DnaD domain-containing protein] [protein_id=WP_042228383.1] [location=2585321..2586010] [gbkey=CDS]
MKPNDNDRKRVERALLAGMTAGTVAVPHLLLRYYHRMKLSELEVMLLIHILAFQENEKKDFPTHDEIQDRMTTPQDQVIE
AIQKLVNSGVLRIEQEIDSVSDIQYERYNLRPMYEKIAGCYAEDLLQKAARQAEAQKPEAQPGREKDVYSIFEAEFARPL
TPMELETITSWLDRDGYKEELVLAALKEAVFAGKVHFRYVDRILLEWSRNRITTVEQAKEFTQKFRGNR
>lcl|NZ_CP026520.1_prot_WP_042228381.1_2337 [gene=mtnB] [locus_tag=PC41400_RS11970] [protein=methylthioribulose 1-phosphate dehydratase] [protein_id=WP_042228381.1] [location=complement(2586109..2586747)] [gbkey=CDS]
MTELTLEQKQHAYDELREIKANLAARGWFPATSGNLSVRVGEFTPDKFTFAVTSSGKDKSLQSPEDFLLVNEKGTPVEAT
RLKPSAETLIHCEIYKYTGAGAILHVHTMFNNLASELYGEEGFIPVEGIEIIKALNIWEEDATIRIPVLPNYADIPKIAE
LVQGAIVPRIPGIVLRKHGIYAWGASVAEAKRHLEAFEFMFEYVYRWHLLKK
>lcl|NZ_CP026520.1_prot_WP_042228380.1_2338 [locus_tag=PC41400_RS11975] [protein=2-hydroxy-3-keto-5-methylthiopentenyl-1-phosphate phosphatase] [protein_id=WP_042228380.1] [location=complement(2586744..2587412)] [gbkey=CDS]
MSTRQKIVFCDFDGTITVSDNIVAIMKHFDPPGWEAIVKRLVEERAISVRHAVGEMFRLLPSSRQQEVIQFAISNAVIRE
GFSDFVAYCKQNGIKLLVTSGGIDFFVYPLLESFDIPREHIYCNGSSFEGETIEILWPHACDEHCGADCGMCKTSIIRSY
PADDYERILIGDSVTDFQGAKLVENVYARSHLIELCEELKLSYVPFETFHDIIRDLEATIQA
>lcl|NZ_CP026520.1_prot_WP_042228379.1_2339 [locus_tag=PC41400_RS11980] [protein=2,3-diketo-5-methylthiopentyl-1-phosphate enolase] [protein_id=WP_042228379.1] [location=complement(2587409..2588713)] [gbkey=CDS]
MSSYCLATYRCYDDTADFNKKAQGIAVGLTVGSWTDLPEARKAQMQQHLGHVVSVDVSEAGEASPESRRYADIRIAYPDV
NFSRDLPALLVTVFGKLSMDGKIKLTDLSFSESFLSAFAGPKFGLQGVRELLGVHDRPLLMSIFKSVIGQDLEGLGEQFY
AQAAGGVDLIKDDEILFENPLTPIEKRVAVCMEASRRAYDVTGQKLLYATNLTGPTSKLADQARKAIHAGANALLFNVLA
YGFDTLCELSADPDISVPIMAHPAMAGAFYPSPEYGISASLLLGKLMRVAGADLVLFPSPYGSVVMPREENLAIKQVLLT
ENLEQDYLYRHSAAEGPEREPSASANVRAALKTSFPVPSAGIHPGLVPLILRDFGTDVVVNAGGGIHGHPMGAIAGGQAF
RQAIGATLDGKPLQEAAQQHPELKAAIEAWGVKS
>lcl|NZ_CP026520.1_prot_WP_042228378.1_2340 [gene=proC] [locus_tag=PC41400_RS11985] [protein=pyrroline-5-carboxylate reductase] [protein_id=WP_042228378.1] [location=complement(2588756..2589610)] [gbkey=CDS]
MNSIIHTKRIAFVGAGSMAEAVLRGLIERGITTPGQVFASNRQNEERLQELRERYGVEASCDPVVKRERISSADIVVLCM
KPKDVEASFDELKPMLNGSQLLVSVIAGLTIRKIEILLDTQMPIVRTMPNTSSTIGLGATGMSFSESVSEEQQELATKMF
EAVGMVTAVPEDYLEIVTGLSGSGPAYVYYFMEALIQAGIAGGLDENEARRLTMQTLLGAVHMVERTGEEPAELRRKVTS
PNGSTLAALQTLDRYKFPEAVAAAVFSSAKRAKEMGAEIAPHIR
>lcl|NZ_CP026520.1_prot_WP_042228376.1_2341 [locus_tag=PC41400_RS11990] [protein=glutamate-5-semialdehyde dehydrogenase] [protein_id=WP_042228376.1] [location=complement(2589675..2590922)] [gbkey=CDS]
MSEVRQKATIAKDASSRLGNLTTGQKNTALLRMAEALVAEQDSIIAANAEDLKRGADTGLSTSLLDRLALTPERIESMAE
GLRQIIELPDPIGDQLESFARPNGLRVEKIRVPLGVIGIIYEARPNVTVDAAGLCLKTGNAVVLRGGSSALRSNERIVEV
LQAALAGTDVPPGALQLISDPDRASVDEMLKLNGLLDVLIPRGGASLIQNVVKNATVPVIETGAGICHTFVDESAKPEMA
EKIAINAKVQRPSVCNSMETLLVHRSHAEQHLSVLAGKLSALGVELYGCPQSRELAPQIKTAADEAWDTEYNDYVLNVKV
VDSLEEALAHIRRHGTMHSECIVTENEAHASRFLQEVDAAAVYHNASTRFTDGFEFGFGAEIGISTQKLHARGPMGLPAL
TSTKYRIYGDGQIRE
>lcl|NZ_CP026520.1_prot_WP_042228375.1_2342 [gene=proB] [locus_tag=PC41400_RS11995] [protein=glutamate 5-kinase] [protein_id=WP_042228375.1] [location=complement(2591070..2592194)] [gbkey=CDS]
MQQRIVVKIGSSSLTSDAGGLNPAPIAYFAAELARLQQNGFEVVLVTSGAVAAGFSFLGYKTRPKLLHEKQAAASVGQAL
LMRAYQEAFHQYGIGAAQILLTRDDFTSRKRVSNATATMEELLRRGFIPIINENDTVSVDEIKFGDNDTLSALVANLLKA
GQLIIITDTDGLYTDDPRKNKDARRIERVDEISDELYRIAGGSGSSVGTGGMRSKIEAARIAMRGGIPVFVGRATEPHDL
RLAAEGNGRGTYFSAADHTLSIKKQWVGFHSVPQGKVTVDQGAENALLSGGKSLLPAGVTAVEGDFHPGDVIEVCGEDGS
VIGRGVANYVAWQLQAVMGLGTGEIQRRVEVPRIEVIHRDEWVTLQAYRASQNT
>lcl|NZ_CP026520.1_prot_WP_241688893.1_2343 [locus_tag=PC41400_RS12000] [protein=pyridoxal phosphate-dependent aminotransferase] [protein_id=WP_241688893.1] [location=complement(2592707..2593858)] [gbkey=CDS]
MASLPSQFFAGLVKKANAQIAQGHDVINLGQGNPDRPTPSHIVKSLQESAENPLYHKYPPFSGYGFLKEAIAARYREDYG
VTLDPETEVAILFGGKTGLVEISQCLLNPGDVCLVPDPGYPDYWSGVALAGAEMAFMPLREDNAFLPDYGSLSENDVRRA
KLMFINYPNNPTASTASAAFYDETIAFAAKNGIVVASDFAYGAIGFDGGRPVSFLERPGAKEVGVEFYTLSKTYNMAGWR
VGFALGNPDIIRIINLMQDHMYVSLFGGIQAAAATALTAPQDCVHELRAVYESRRNALFGALNEIGWEAKPSQGSFFAWL
PVPRKYSSAEFADLLLNEAKVVVAPGIGFGTHGEGYVRLGLLTGEERLQEAVRRIGKLHLFDK
>lcl|NZ_CP026520.1_prot_WP_042228373.1_2344 [locus_tag=PC41400_RS12005] [protein=right-handed parallel beta-helix repeat-containing protein] [protein_id=WP_042228373.1] [location=complement(2594077..2595360)] [gbkey=CDS]
MNSSRPLSRRKLLAALGAAGAVMAAGSLPSAVKARGAETLIHNVKEYGALGQGSYGDTDSAGIQSAVHAAAQTGGIVYIP
AGRYILTKPVQLKSGIRIMGAGISLTTLTATAGCAFTSLPPASDIEIASLSFHGGSAPGPVASPVEDSIISLLQASHVSI
TDCRFFTLSSPVLLRNCHSCRVDHCDFELILGNIPGKPHYGTGIYCSGGEGHTFFRNTFKSVTNTAVYLDGGCSRSVIQS
NDVTYIDQYAFLVQSDQAAEPAARIDIIYNRITNLDAAKSGITAILLKRNVSGCTVSHNEITGLDGDGIRLEGDQESGKL
QPADHTLHGNVLKGIKGNGVVISNAVRASVHGGTFRNLGASGIHIESRGETGSRETAASGNLFSGCKTAGIRISGKKSTG
SLLFGNGGTGNGQNVLDEGTGTVKDSF
>lcl|NZ_CP026520.1_prot_WP_042228372.1_2345 [locus_tag=PC41400_RS12010] [protein=carbon-nitrogen family hydrolase] [protein_id=WP_042228372.1] [location=2595572..2596387] [gbkey=CDS]
MDNQQSFPNRVRLALIQMDITIGEPEINRRKVEQLLLEAVGGAQKPDIIVLPEMWNTGYALDRIHELADPEGRITRDMLG
SFAKEHGVHIIGGSVAEKKGEEVKNTVYGFNRNGEEILEYSKIHLFQLMDEHLSLAAGSQRGSFDLDGIPAAAVICYDIR
FPELTRKLALDGAQMLFVPAEWPHPRLHHWRTLLMARAIENQMIVVSCNRVGESKGTSFFGHSMVIDPWGQVLAEGGEEE
EIIFAEVDLAEVASVRGKIPVFADRRPAVYE
>lcl|NZ_CP026520.1_prot_WP_042228371.1_2346 [locus_tag=PC41400_RS12015] [protein=DUF1328 domain-containing protein] [protein_id=WP_042228371.1] [location=complement(2596488..2596652)] [gbkey=CDS]
MLKWAMIFLIIAIVAGIFGFFGIVSAAAGIAKVLFFLFLILFVITLITGRRSSV
>lcl|NZ_CP026520.1_prot_WP_042228370.1_2347 [locus_tag=PC41400_RS12020] [protein=general stress protein] [protein_id=WP_042228370.1] [location=complement(2596761..2597198)] [gbkey=CDS]
METKVRLAANEEQALREVKNLQAEGFNLDEIYVMTHDNDRTEGVAEMTNTNEVGVAEEGLMTAAANLFRSKGDKLRAKME
SLGLSSSEADTYEKELDRGKILVIAYADRDGDGYPHDGRLRGEETTLPPTGIYMSGRDNQNSNLF
>lcl|NZ_CP026520.1_prot_WP_042228369.1_2348 [locus_tag=PC41400_RS12025] [protein=fused response regulator/phosphatase] [protein_id=WP_042228369.1] [location=2597654..2598793] [gbkey=CDS]
MSIMIVDDNPVNLIVIEKILKSAGYSNMHRASSAMELFDMLKIDNPSSSDLQIDLILMDLMMPEIDGIEACKKLQQIEEL
KDIPIIIITAMGDSIKLAEALEAGAIDYVTKPINKIELVARIKSALRLKYEKDWHKENDKRIRSELQLAKNVQSSVLSLP
IETEKIRIEAYYNPSYELAGDLYNWYPIDEDRYGVMLLDIMGHGISSSLVCMFISSVLRDTITKFVTPELVIAELNRYMN
RLHNDQNHLNYYFTAIYMLVDTKNKTIDYVNAGHPPGFLMNDSGKAIPLANGTCAVGFFEEMDIKPGRITYTGKSRILLY
TDGLLEEIEEAETELPTRLITAVQQKGDMGLHAVVDELLDEEVQNNHMDDICLIWIDIP
>lcl|NZ_CP026520.1_prot_WP_042228368.1_2349 [locus_tag=PC41400_RS12030] [protein=response regulator] [protein_id=WP_042228368.1] [location=2598843..2602520] [gbkey=CDS]
MKISTKLMAGFGLLFILMFVLAGIGTTRISSIDESLDSAFVNRFNKTRLAYTARSDVNSIAKYMANLLLNSDPNQPGAKE
NLAKANTYKEKLKEGVAALQKSADSDVEKQLTYETVKSTENFMSYLNQAIELYDNGKVAEANKLRTDIGVSYQDQVTESV
DHLTQYLDDNVQKGINEFKASNSKTLLLTILLTVAALVIGLGIMYWIMRSLSRGLALLTGMIKGFGNGRFDPGYRVEVTT
TDEFGKVAEVFNVMADDLEELMHNERTFNKQNEDQAWLKSNVANMSMLLQDQTNMHDLTNSFISEASRMIGATFGAVYLL
EREGMRTRLVLSSAYAFDKDPEDNLLPEFALGQGLVGQAAADGRRIMVQDEVDTYLTIQSSFGQLDAKYLVVEPVVSEEE
VGAVVEFAGLKMLSENELDLLEQLTENLAVSIGRIRSRERVEELLRMSQALTEELQSQSEELISQQEELRTSNEKLEEQT
KALKASEELLQSQQEELEQNNEELLRKTKQLQTQVQETELANRQIEEAKEALEKQAFELAMASKYKSEFLANMSHELRTP
LNSMLILSQMLAENKDSNLYDKQVEYAETIHSSGSDLLKLINDVLDLSKVEAGKVEVQAEYVLVPDIMETIQRSFEQVAR
KKALGFEITIDSGVPRMLYTDSMRLEQILRNLLSNAFKFTSQGSVLLHVYPVKEGEPQIAFSVKDTGIGIAEEKQQIIFE
AFQQADGTTSRQFGGTGLGLSISKQLAGLLGGTIEICSREGEGSTFTLFIPHRKYSERVGGSAEAAAAGQEQQALPLPDA
PAVPQLPAPAPVPVKPVPKAAQREVEDDRGSITENDKVLLIVEDDVHFMNVLVDMARSRGFKALVAMQGDIGLQMAKEYK
PDGIILDIQLPIVDGWSILVQLKNNAETRHIPVHVMSVVDEVHQGLSMGAIAYMRKPSSKDVLEQAFIELEGFIDQNLRK
LLLVEDDPVQMNNLIELIRHDDVLIEVATTADEAIVKLRERHFDCMVLDIGLADHSAFDLLERIKAEEELRRLPIVIYTE
KEYDKKDEMRLKKYAESIIIKNVKSPERLLDETALFLHRVEETLPEEKKRILQRLHSSEAVFAGKKVLLVDDDIRNVFAL
SNLLEGYNMKVLFAETGRQALSILEKEPDLDLVLMDIMMPEMDGYEAMKEIRSMPQFDNLPIIALTAKAMRDDREKCIRA
GASDYITKPINNDQLMSLIRVWLFK
>lcl|NZ_CP026520.1_prot_WP_042228366.1_2350 [locus_tag=PC41400_RS12035] [protein=protein-glutamate O-methyltransferase CheR] [protein_id=WP_042228366.1] [location=2602607..2603476] [gbkey=CDS]
MDSNFHPIDQLPTFSGLQAEHERIEISLLLEGIYRIYGYDFRNYSYPSIRRRIWHRILAEKLTTVSGLQERVLHDRECLE
RLLSDLVIHVTEMFRDPEVFLSFRQKVVPLLRQYPAIRIWHAGCATGEEVYSMAILLHEEGLYDRTRIYATDINEDVLKK
AEKGIYPLRSMKDYTRNYLQSGGKEAFSKYYTASKDSVTLHSFLRDNIVFAQHNLVTDRSFNEFHVILCRNVLIYFNTQL
QNQVHTLIYESLGPSGFLILGDKETITFTKHASAYEAVSTADKIYRKVQ
>lcl|NZ_CP026520.1_prot_WP_084706558.1_2351 [locus_tag=PC41400_RS12040] [protein=ATP-binding protein] [protein_id=WP_084706558.1] [location=2603523..2604707] [gbkey=CDS]
MNLDEPIKILMVDDQPENLLALEAVLAEENYHLVKAGSGEEALRCLLHDDFAVIVLDVQMPGMDGFETARWIKSREKSKD
IPIIFITAASHEKEHVFAAYSVGAIDYIVKPFVPHTLKAKIEGFVNLYLVQKKLQHQSELLNMRTQELLTAKEAAESANR
AKSEFLALMSHEIRTPMNGVMAMTDLLLETELSDEQREYADTISRSGAALLHIINQILDFTKLESSKMEMNTEPFDLDVC
MEETMDLFYGECRKKDLNLEREIAENVPTHLIGDELKLRQVLINLIGNAVKFTETGGIKVQVTNLGKAGDGSRIILQFRV
RDTGIGIPAEKRNRLFQPFTQVDSSMTRKYGGTGLGLVICQNLVELMSGKMEVYSQEGQGTEFVFTVQAEELRV
>lcl|NZ_CP026520.1_prot_WP_042228365.1_2352 [locus_tag=PC41400_RS12045] [protein=glucose-1-phosphate adenylyltransferase] [protein_id=WP_042228365.1] [location=2605047..2606228] [gbkey=CDS]
MRKQSVALLLAGGEGRRLGALTQNTAKPAVHFAGRYRMIDYALSNCRNSGVDTIGVVTQYKPESLHQHIGTGEAWLSETD
KGCVEVLCSNQEHVIGNEFTGTANAVYQNRHFIEKHDPEYVLVLSGDHIYQMDYSKMIEQHQKLGSVATIAVKPVGWDEA
SRFGIMNTDEEGRIVEFEEKPAQPKSNLASMGIYVFNWDFLKKVLERDDLSADSSHDFGKDVIPMMVEEGVRVHSFSYEG
YWKDVGTIDSLWQAQMDLLNGSPLMSELEWPMYTARVANAKPHVDRTAKVDRSLISENCAVYGQVQNSVISHGVYIGKGS
VLTDCVIMPNVRIGNNVKLHKAIIGEGSVLCDGVTVGKTDAQTISVVGDHEVLYKINDKAPKIYLPRLEQRIG
>lcl|NZ_CP026520.1_prot_WP_042228363.1_2353 [locus_tag=PC41400_RS12050] [protein=general stress protein] [protein_id=WP_042228363.1] [location=2606242..2606634] [gbkey=CDS]
MNHSIIVVDSEKEIRDVIRQLQTAGCEPHCIHVLSDEAVDTEEMLRPRYTSRIGFSAEQMAYTMATVFRGSGMAMRSKLI
SLGLAGSAIPYYEEEISRGRIVVVAVPSQFVYAYDASYADQAFEASAAGQ
>lcl|NZ_CP026520.1_prot_WP_042228361.1_2354 [locus_tag=PC41400_RS12055] [protein=hypothetical protein] [protein_id=WP_042228361.1] [location=2606740..2607375] [gbkey=CDS]
MNKKSLTMALLGGALLFGGAHGLEGPASAGADDGLISQAAPPLVLGAGGAITEIAGYKTANANGTVSAMPAVAPAEVSPV
GTSGSIKLERVNGISLSDDVNAIVKRKGEPKSIRKDEFFGDTKILEYPDCRVAVGEYGVEYVEVTPDHKQINLDGKPVAN
NLQALEKTLGEPYWVAEDGLVYTHDSQALKLFVDPNSGQLKSVHFFSSVTE
>lcl|NZ_CP026520.1_prot_WP_164977122.1_2355 [locus_tag=PC41400_RS29750] [protein=hypothetical protein] [protein_id=WP_164977122.1] [location=complement(2607779..2607937)] [gbkey=CDS]
MDQDKNVDIEQRHIEKRSDSSAVASTFIKYAAYLIIFFGFLYFLIKYVFPKF
>lcl|NZ_CP026520.1_prot_WP_042228359.1_2356 [locus_tag=PC41400_RS12060] [protein=Dps family protein] [protein_id=WP_042228359.1] [location=complement(2607978..2608439)] [gbkey=CDS]
MAKTSTAKEKQTVTLLNKQVANWGVMYVKLHQFHWYLKGPSFFVLHVKFEELYKEADLYLDTIAERILGLQGSPLSTTKE
FQTNSSIKEVSPERNAGQMVEELIQDFRTITKESKEAIETAENEGDPGTADMLTEICTNLEKHIWMLESYIND
>lcl|NZ_CP026520.1_prot_WP_042228358.1_2357 [locus_tag=PC41400_RS12065] [protein=DUF948 domain-containing protein] [protein_id=WP_042228358.1] [location=2608634..2609092] [gbkey=CDS]
MIWQISLAVIAVAFVVLVVFLVKTLRSVTDLVAQTNQTIQQVQQQVTVVSQEANELLRHTNEVSLDVRNKLHALDKTFYT
IKNVGDVVSEITTSVRQTSATVTNTMRNKVEKELNSPKSIVNRIAPLVPVAVDMWKRLKQSRTTNSKSVAVK
>lcl|NZ_CP026520.1_prot_WP_042228357.1_2358 [locus_tag=PC41400_RS12070] [protein=STAS domain-containing protein] [protein_id=WP_042228357.1] [location=2609241..2609576] [gbkey=CDS]
MVQENKFRMEKQITDDDNIISLIGELDLSKASEFNTNVDPFVQDVGRKLILNLKELTYIDSTGIGVILTIIKTRHALKAS
FAVENIPPKVQKLFDLTGLTPFLSAVANTVN
>lcl|NZ_CP026520.1_prot_WP_042228356.1_2359 [gene=rsbW] [locus_tag=PC41400_RS12075] [protein=anti-sigma B factor RsbW] [protein_id=WP_042228356.1] [location=2609597..2610046] [gbkey=CDS]
MESKNRTVSLTLPAEAEYIDLVRLTLYGISSKMGFSYEDIEDMKVAVSEACNNVVVHAYEPGQGGKMEVHFEEVEGGIKI
IVQDQGLSFDHVTKVGSADSLHNKSLDEVSSGGLGLYLMQALMDHVEVKGDHGTVVILTKLVSKSEEMV
>lcl|NZ_CP026520.1_prot_WP_042228355.1_2360 [locus_tag=PC41400_RS12080] [protein=sigma-70 family RNA polymerase sigma factor] [protein_id=WP_042228355.1] [location=2610043..2610822] [gbkey=CDS]
MKLQSERKEMEHIHDLIRQYQETECNDIATSLLIQYESTVKMAAAKMSRNRPDLYEDLIQIGQISLLKLFKQFDPSHGVQ
FDAYMMKSIIGHMKNYLRDKSWYIQVPRRIKEKGFQIQQAIDHLTMRLERSPNVNEIAEELGLTVEETIEILAGRECYHY
VSLDMPLSTEENGSTIGDVLGTQVDEYLNVDNKLDLQEALSHLKPEEQKVLMLAYVDGYSQRTIAQDLGVSQMSVSRIQR
RALDKLRKYFEKNKPGEGA
>lcl|NZ_CP026520.1_prot_WP_042228354.1_2361 [locus_tag=PC41400_RS12085] [protein=magnesium transporter CorA family protein] [protein_id=WP_042228354.1] [location=2610916..2611857] [gbkey=CDS]
MEDKELESSSNWVWHDLHDKEGDLNVAVDALPEELKPLVKQAAAEPFEARLYRHKGSELLEGAFYYDFDSKCREGTPIFH
FLVNEHHLVTINFKEELLSPFDPDLIREQALNAKNALEGLFVILGYIAERLLHVLTEYEKKMRQVGEHMRKRNNSKLLDR
ILDRRHELLDLTKWVRPLKDTMMTTKELFVENMEEGLEHRRMCIKIERIMTRLKLYTEEVEALLALDTSIANYRGNEITK
TLTVFTAVFAPPTMLAGIWGMNFENMPELKWTAGYYLALGLIVALPIVVLGWLHWKGWTGDIIKGKNKNSNLK
>lcl|NZ_CP026520.1_prot_WP_042228353.1_2362 [locus_tag=PC41400_RS12090] [protein=hypothetical protein] [protein_id=WP_042228353.1] [location=2612005..2612319] [gbkey=CDS]
MTVKYTIDCEGSEILVREVDGATYQLCIRASTNPLGNGNVLETYSEFYRAVRAADHFCETYAKAKENGFYLQETFLVKPD
KHPIPVSKILNEEMQSEQVNALFA
>lcl|NZ_CP026520.1_prot_WP_042228351.1_2363 [locus_tag=PC41400_RS12095] [protein=DUF5665 domain-containing protein] [protein_id=WP_042228351.1] [location=complement(2612421..2612750)] [gbkey=CDS]
MRHSSPTRERRVPGEVPIPSQEPSRKELALERLATILERSEYKDVLENYLNPRKRIITNFTAGLARGLGLTIGTTVVVAL
LGWIVSQFLSMPIIGEFIAKLVSYVDMYK
>lcl|NZ_CP026520.1_prot_WP_042228349.1_2364 [locus_tag=PC41400_RS12100] [protein=YtxH domain-containing protein] [protein_id=WP_042228349.1] [location=complement(2612880..2613245)] [gbkey=CDS]
MSNESKGNGVLLGTLVGAGVGAITALLLAPKTGREMREDLSAKYSEWMAKGQELASDAKTKVADVASRVNEQTSNLVDRS
KDLVNSAADSTNRALDNVKEMKEERTDQMLNSMDSSNKPMN
>lcl|NZ_CP026520.1_prot_WP_009675502.1_2365 [locus_tag=PC41400_RS12105] [protein=GlsB/YeaQ/YmgE family stress response membrane protein] [protein_id=WP_009675502.1] [location=complement(2613325..2613585)] [gbkey=CDS]
MIGFIITIIMAVVIGIIGDAIVRSNMPGGILGSMLAGFVGAWLGSLLLGSWGPEIGGFSVIPAIIGAALFVFLIGLASRA
FHRTNA
>lcl|NZ_CP026520.1_prot_WP_042228348.1_2366 [locus_tag=PC41400_RS12110] [protein=LysR family transcriptional regulator] [protein_id=WP_042228348.1] [location=complement(2613687..2614601)] [gbkey=CDS]
MELRQLQYAVQIALERNFSRAAEKLHIAQPSLSQQLSKLEREIGVLLFQRSTNSVELTHAGSIFVDKAQAILDSLEQLKQ
EMEDISQMKKGKLVVGSMPITGSTIMPYVLPVFQSAYPDIEVVLVEETSANLEKLTTSGQTDISLLSLPLQEDSLVYSHL
LEEEIVIAFPPSHPLAKRHMENGEPVTISELENEAFIALKKGQGFRKINVALCQEAGFTPNIVFESSNMETVQSLVAAGM
GIAFVPYLIAKRTRSELSPIHLPLRNHPSRTLVIAHRRGRYLSKAADAFIETIQAVMKEQGSFE
>lcl|NZ_CP026520.1_prot_WP_042228346.1_2367 [gene=leuC] [locus_tag=PC41400_RS12115] [protein=3-isopropylmalate dehydratase large subunit] [protein_id=WP_042228346.1] [location=2614731..2616152] [gbkey=CDS]
MSKKTLFEKIWNNHVIYQEEGKPSILYIDLHLVHEVTSPQAFEGLRLSGRRVRRPELTFATMDHNVPTKDRFNIKDPISK
QQIDTLSQNCRDFGVTLFDLNHVDQGVVHVMGPELGLTHPGKTIVCGDSHTSTHGAFGALAFGIGTSEVEHVMATQTLQQ
AKPKTLEVRVNGALPVGVTAKDLILGIIAQYGTDFATGYVIEYTGEAIRGLSMEERMTVCNMSIEAGARAGLIAPDETTF
EYLRGRQYVPQDAAFDQSVEAWKLLTTDEGASYDLVVDFDAASLVPQVTWGTSPGMGTGVTSSVPNPQEMTTENERKAAE
KALEYMGLEPGTPMTELAVDRVFIGSCTNGRIEDLRRAAAVASGYKVNANVNAIVVPGSGRVKLQAEKEGLDKIFIEAGF
EWRDAGCSMCLAMNPDVLSPGERCASTSNRNFEGRQGRGGRTHLVSPEMAAAAAIEGRFTDVRGWTYRVPQEA
>lcl|NZ_CP026520.1_prot_WP_042228344.1_2368 [gene=leuD] [locus_tag=PC41400_RS12120] [protein=3-isopropylmalate dehydratase small subunit] [protein_id=WP_042228344.1] [location=2616188..2616796] [gbkey=CDS]
MEAFKQHTGLVGPVDRVNVDTDAIIPKQFLKRIERTGFGQFLFFEWRFTEDGQVIPTFDLNKPRYQGASVLISRANFGCG
SSREHAPWAILDYGFKVVIAPSYADIFYNNCFKNGILPIKLSEEQVEELFQCTAKHEGYKLNVDLENKTLTDEYGLKIDF
DLDEHRRQFLLQGLDDIGLTLQHVDKISAYEAQHQERLALNL
>lcl|NZ_CP026520.1_prot_WP_241688844.1_2369 [locus_tag=PC41400_RS12125] [protein=N-acetylmuramoyl-L-alanine amidase family protein] [protein_id=WP_241688844.1] [location=2617095..2618507] [gbkey=CDS]
MMKLWSTVATLLMIALLLPSLAFGATSPPTAKIVLDGKTLTTEVDPLNVDGNVLVPIRQIFEKLGAKVTWDQSKQTATIV
RDSMKIQLTVGDRNPVVNGSRAEQLEVAPMNVKGNVLVPVRFVGQKIGYDFKWVQATSTVQIFPKSGQGSGAGTPPTTPT
PTPTPTPNPGQVSDKVLVKTIELTKDQLLVGTDKKSPQANVFKLTSPDRIVIDLPGTELDASLAAKLVNNVGEAPSLHPM
VEKVRFSNFDNDPATVRITLDMKAKADLTISAAAGQANLLVGVLKAPEPGKTKYKVVIDAGHGGTDSGAVSLTNKKEKDF
VLAVANKVAKLLEKEPNIQTYMTRSTDTFVTLKGRSEFANNLDADLFLSIHGNKATPTAGGTETFYYERADSKAFANVVQ
KHLLEATGFKDRGVKEGDLHVVRETKMPAALAEVGFLSNAQEEALMYTSAFQDKVAASFVAAIKEYLKIS
>lcl|NZ_CP026520.1_prot_WP_042228340.1_2370 [locus_tag=PC41400_RS12130] [protein=GerMN domain-containing protein] [protein_id=WP_042228340.1] [location=2618529..2619113] [gbkey=CDS]
MKKKLWVQGIVVGSLVLAVASGCGQKQQPLASNNGGSTPEPVQTVSPSPEPTISVKPSPSPTPEAKQTEIKAYYSDPDLT
KLVEKNVKISYKTEGDKYKSALEALKKSDDAQAVPLFKDVTFQSVKFEKDELKLDLKMGESAQLGSGGEQFFLDALKKTV
FQFPEVKAIYITKDGGQVESLMGHMDLPYPIKRK
>lcl|NZ_CP026520.1_prot_WP_042228338.1_2371 [locus_tag=PC41400_RS12135] [protein=S-layer homology domain-containing protein] [protein_id=WP_042228338.1] [location=2619152..2621839] [gbkey=CDS]
MKTNKTAMLLAGALIASSISLSPAYADNKTTDTSTSTQAALAAELKKNFPDVVNTHWALRHITKLALEGIVEGDEKGNYR
PEASVSQQDVIIMAVRTLGLESSISENTSYNLPLKVDTYAQKYVAAAIEKKLISVDEEKDGSKNWGSKPASREWVAKLTI
RAIGQQETANGLTDDTPAFTDLKSISSWAKPYISAATVLGIVDGFEDGTFRPKGEVTRAQMATFLSRAERHSKNLSTRVV
RGYVNHVSADKISIFDEKGKTTSYKLTPESVFYGNKNDNPIQPSELAYTYQVSLVQQGGTVYYAEILKDEDRSEKIKGTL
LNVKDRKLTLDNFEAYEIGSGALVLDAKGQEISLSSLVVGSKLELRRNSITGSKEYGQVVVREVPVNKTSDGTVQSVNGD
TVAVVDLTSGQSESYTLAKNPTIVDADSRLVDKTAIKAGDAISYTIENSQITQVKIKTRTGVSKTVKGNFSSLSSDKTGI
FYKETDGSFGAGKLSANVQVVIDGVGSPSLYDLDENDELSMDIANDQIVKITVTNRSISNLLFATFIRFDSKYNQLNVQH
ADGTPGAYTISDNTVVTYGGTSMSFKNYPGFASLFTEGKKVDLKVSKNKVTAIQLSLDVSGTITQINTVANEMSIRANNG
QSFTFKMYSPAVEIPNKSGATLADLKIGDLVTASLDYNQAGLTKVVVDRSTLVKVQYVNSSQQVTAVDELNNGVTFTLKD
TKIVKSDGTSATATDILPDDYLYVTYKGGSVVKAEIVSPSRGKVTNVDSVNGTLTIQEPTGKIQLYTIGTDAVVKNGTTL
IGLNAVKAGDRVQVIKQSNGKINVQLAAAMTKTFSSYEYAVNQIQFEAGGSIERYNIHRSAYYHKGTETLTASSFNKGET
VTVYIVDNKIIEMEK
>lcl|NZ_CP026520.1_prot_WP_042228337.1_2372 [gene=nth] [locus_tag=PC41400_RS12140] [protein=endonuclease III] [protein_id=WP_042228337.1] [location=2621993..2622652] [gbkey=CDS]
MNQKVRHILDTIGEMFPDAHCELNHSNPFELTIAVLLSAQCTDETVNKVTQTLFQKYKTPEDYLAVPLEELEQDIRRIGL
YRNKAKNIQKLCQLILERYDGQIPTTHEQLVELPGVGRKTANVVVSNAFGVPAIAVDTHVERVSKRLGLAGKDDSVLEVE
KKLMRKVPREEWTLTHHRLIFFGRYHCKAQSPKCDICPLLDICPEGKKRMKIKKVSKTS
>lcl|NZ_CP026520.1_prot_WP_009675599.1_2373 [locus_tag=PC41400_RS12145] [protein=hypothetical protein] [protein_id=WP_009675599.1] [location=2622682..2622960] [gbkey=CDS]
MKCIAVYTKDFELFSDLYETIMNTPLSENEEKEIQGVIVSESGEVPENYLEKMRTKPEVVVMKVKEGDITILQHGDVFEI
FLPEEKESAVVS
>lcl|NZ_CP026520.1_prot_WP_042228336.1_2374 [locus_tag=PC41400_RS12150] [protein=dynamin family protein] [protein_id=WP_042228336.1] [location=2623162..2626944] [gbkey=CDS]
MNGLMEQRKDEVLRAFDRLREEMKGAGDGARVRKLIQLEDKLGGEKTVLAFCGHFSAGKSTLINTLCGHRLLPSSPIPTS
ANIVSIESGEEGAEVVQRSEGEEEARKRRIPLSELDAYCVNGTDIEEVRISYPLPFLGEHAVLLDTPGIDSTDGAHHMAT
ESALHLADAVFYVMDYNHVQSEVNLAFTKQMKERGKPLYLIVNQIDKHREEELSFDTYKESALEAFRSWGIYPDGMFFVT
MKQPDHPRNESGKLKWLLSRLIGMQEELREASVDAAADHLIGEHIEWLREQSEPQKAVLRERVDEEESAGVLAEFQELTR
RLNEAQSLPGHLTDSLRKEIAAIIDNANVTPALTRDKAHEYLLSRKPGFKVGFFGAAAKTAAEVERRLDVFREDYAEQVH
AQMDWHLRDALRRAYSGIGMTDEKLLADIDAFSVEVTSSWLASQVNTEAVFGNEYTMTYMRQTAAELKTAARRSALAMID
RLAGLAGSCSVRLDSELAPDLRQLEERLGALRELKRFEREEAAREEALRALASWPHAKAFELPDLAAYRGEEAAVSGVLH
TSEMETILRAAHTAPGDRDSLADGEDGAAAAPQNAPAAGASAAPAGGNAASGAAAVAADGLQLGGAKRAERRAQDLAAAA
DLLGDMPSMRSIVRSLREKAERLRHKTFTVALFGAFSAGKSSFTNALIGKRVLPVSPNPTTAAINKIVPPTADWPHGTAK
VQMKTAEVMLQDVLFSLDVLGVRASAMEEALAAIRGLTPESVTAKGKPHFTFLKAVAKGWPDAAPLLGTQQRVGEDAFAG
YVAEESKSCFVEYIELHYSTPLTEQGIVFVDTPGADSINARHTGVAFDYIKNADAILFVTYYNHAFSQADREFLLQLGRV
KDSFELDKMFFLVNAADLAASEEELQGVVRHIETNLLQHGIRHPRIYPVSSHLAAEGKLSRDNNKLEASGIVPFEQDFIR
FTLEELTDISLRSADQELERALHVLQERIDGAQQGEEARKARLQALERAEEAAVRLLKDMDPAPVQRELAKEIQELLYYV
KQRTMYRFGELYNASFNPASFREGGGDPKAALRESWRDLLRMIEFDLQQEVLATSLRIEQFLNKLAKGTYESWGTGLRKL
LESYEFEAYESVSFDTPDNKLPLEAPDVTDKTLAVHFKNAKHFFEGEGKSKLRAELEGRLNDPVALFVENAVTMLKVLYA
GHLSDTFAALEERHLESYREHLEGMRDALQMRTDVGVLASKRDELAKWIPSRSGEASAGH
>lcl|NZ_CP026520.1_prot_WP_042228334.1_2375 [locus_tag=PC41400_RS12155] [protein=ABC transporter ATP-binding protein] [protein_id=WP_042228334.1] [location=2627582..2629366] [gbkey=CDS]
MEVFKGLKRFYWPERKLLFASIFCLICTTALGLVYPNLLRYLIDDVIAVKRYDQVPLLALFVVGVIAVKALFQFLHGFSG
GRLGNRVAYNMREAMYEKLQYLSFQYYDKARTGDLMSRLTADLEAIRNFIGFGFAQILNVFLMVIFGMVMMFSISWKLTL
LTMLTMPFLAFAAIRFEGKIHPAFRSLRASMSNMTTAVQENITGVRTVKSFAREPYEVEKFTLRNEEYRANNVITSEIWS
KYFPVMELIANLSAVILLGVGGYMVILGSLSVGELVAFFSLVWYIIGPMWGIGFHINNYTQSKASGEKILEILHQYVHVK
DGEHAVELNPATVKGHVCFERVTFAYPDAEAAIHDLSLNAPPGTVIGFLGGTGSGKSTIISLLMRAYNVKTGQITLDGMD
IKDIRLESLRSQIATVFQETFLFSSTIKGNIAYGRKDVTMDEIIRAAKLVKAHDFIMEMPLGYDTVVGERGMGLSGGQKQ
RIAIARALIKDPKILILDDATSAVDMETEQEIQAGFKEVMRGRTTFIIAHRISSLKHADEIIVLDKGRVVQRGTHSELIR
REGPYLETYRIQYSDHPDVAGAADAQQRRAAVHE
>lcl|NZ_CP026520.1_prot_WP_042228333.1_2376 [locus_tag=PC41400_RS12160] [protein=ABC transporter ATP-binding protein] [protein_id=WP_042228333.1] [location=2629359..2631209] [gbkey=CDS]
MSNRQRQSEEKVNGKGAGSGERFVYQDDEMLEKSFDWGQFKRLAVYMKPYSKQIIPIVIVMMIVGAITKLSIPLIISTTI
DSSLHVGGDAKFLFGMVALMLAVYIIQWLSNKYRIKYTSVIGQRVIYDLRHDLFRHIQRLSFEFFDKRPAGSVLVRVTNY
VNSLQDLFTNGVVNMMIDCVQLVGIVIILLTINFKLGAAIIITVPLMFIVSSQLRKRVRRSWQTVNIKQSRLNAHLNECI
QGIRVTQAYTQEKENIRFFEQMNEDNKQAWNRASKLNQSFGPIIEITGAIGYFILFWYGASLIQSGEITVGLLVAFASYI
GNFWEPINRMVQMYSQLLVAMASAERIFEFMDEQPTVGEKENAAELPPIRGQVSFENIVFEYEKGRTALNHINVDVQPGQ
SIALVGHTGSGKSTIINLLCRFYDPTEGSVRIDGKDIRDVKLQSLRSQIGIVLQDTFIFSGTIRDNIRFGRLDATDEEVE
AAAKAVHAHDFIVKLPFGYDTEVQERGNALSMGQRQLISFARALLADPAILILDEATASIDTETELRIQEALKTLLKGRT
SFMVAHRLSTIRGADKIIVLDHGRIMETGDHDELMKHQGIYYGLIQAQYKFLQDAI
>lcl|NZ_CP026520.1_prot_WP_042228332.1_2377 [locus_tag=PC41400_RS12165] [protein=acyltransferase] [protein_id=WP_042228332.1] [location=2631376..2632512] [gbkey=CDS]
MNSQITTKTSKLKIFEMDIVRAVAIMAVVLIHSTTEAAVYPWEGQVVPDEGSLSQIILFTLNRACQFAVPVFILISGLVL
FYRYSGSWSVKTAVQFYKRRLWSVVVPYLIWSLFYYIYNPWISGQPVELDAREFLGDIQWADTGYHLYFMIIILQFYVVF
PVLMFLAKYKWFRVSMVPLGILIQLGFYSYNHWVERIDHYPSLAPTYFSFFLTGGFIGLYYDFFRQKMKSVVSWALLVML
VSGGTYVEMFLLNRYDNRMFENTWYVIAMLFYATTVSVVLLGVGKWLLDKLPGAAKWLLPLGSYSFGIYLMHPLILTAFK
LKIPAPGGMAAYNVYVLLSFLSALLGSLLAAVLYRKAARALKPGKSKPASPGNVPMNG
>lcl|NZ_CP026520.1_prot_WP_042228330.1_2378 [locus_tag=PC41400_RS12170] [protein=hypothetical protein] [protein_id=WP_042228330.1] [location=complement(2632578..2633261)] [gbkey=CDS]
MRLVKVRNRLTGLLMLALLAGGCSAPAVSPPSVSPSPLPSAAGPEQLIVEPADLFAGDAAKFSPFLGSYAGSVFLRYKGD
KKVLSVSYKVWENGKASESAVSKHSLPMMNGDATVPKDYNGEFIISVREVTDDKDESILNSYTVTISLKDSGGSTTSSFT
VPVNRKPFTQSHLLKLQEKITIPDSTPTAVWGFAASNANAVTFTSGKLPEEEASKAPYAFVMYLTLE
>lcl|NZ_CP026520.1_prot_WP_042228328.1_2379 [locus_tag=PC41400_RS12175] [protein=GDSL-type esterase/lipase family protein] [protein_id=WP_042228328.1] [location=2633517..2634326] [gbkey=CDS]
MKSRTLWFVVGTTGLISTLLFAGGFGYAVKDVLSPGSGGEQLTAAPKETAPVNELDRKNKIQLVSLGDSLTAGTGDGTGK
GYVGRFREKLEEQTGKPVYVLNNLAIPGATTDALLKEWDGKKTTDALKQADLILLTIGGNDIFQGGSDIFTNNDQGFNPE
AAAKRVEPTLANLDKILTNINKANPDAKVLYVGLYHPFLDLDPKKAGSLIVQKFNDGAFALTNKFPNAVFVPTYDLFEQN
LNKYLFTDHFHPNQDGYERIASRMAQILK
>lcl|NZ_CP026520.1_prot_WP_042228324.1_2380 [locus_tag=PC41400_RS12180] [protein=ABC transporter ATP-binding protein] [protein_id=WP_042228324.1] [location=2634342..2635328] [gbkey=CDS]
MAGDRERVLSVRDVRKSIRGKEIIKGISFDVYGGEIFGFLGPNGSGKTTTIRMLVDLIKPSSGVVTICGFDVQKQHNEAM
QYVGCIVENPELYGYLTGWENLEHFARMLPDTGEERIREVVELVGMDARIHDKVKTYSLGMRQRLGIAQALLARPKLLIL
DEPTNGLDPQGIKEMRIFIRKLAKGGLSLFVSSHLLSEIQQLCDRVAIISKGQVIAVGEVDELIVQSGSTVLWSVDPADE
AEAWLGRQQGVTVLGRREELSEEDEPAVLSSLHAGERAGTIVTQLSEERIPEINRGLVDAGIRIYAIEMKHPTLEDLFLS
LTEGEKIG
>lcl|NZ_CP026520.1_prot_WP_009675515.1_2381 [locus_tag=PC41400_RS12185] [protein=ABC transporter permease] [protein_id=WP_009675515.1] [location=2635321..2636313] [gbkey=CDS]
MGNLAALVQNESIKMVKKRRFLVVILILLALIPMFTYAQMRVSQNTIKQFGTTDWRASERQKIVDYQRSLSSARVPEEWK
EFRRAQIKISQYYLDHDINPQSPNGVTFTREFLNNSISLFIPLIIMVIAADIISSEHSGGTIKLLLTRPVKRWKILMSKL
ITLILFVSLVVLSTAALSYLISGLVFGYGGWNMPVFVGFKTSGGDVDLSGVHAIDQWKYLLMETGLVWFVGIVVAILSLM
LSVLIRSTAAGMGIMLAVLISGTILTNMVSSWEAAKYLFMVNLKLTDYMTGALPPIKGMSLGFSLAVLSVWAVISLFVSF
GVFTRKDILN
>lcl|NZ_CP026520.1_prot_WP_042228322.1_2382 [gene=parE] [locus_tag=PC41400_RS12190] [protein=DNA topoisomerase IV subunit B] [protein_id=WP_042228322.1] [location=2636362..2638335] [gbkey=CDS]
MTEQLEMMTNQPNQSAEYDADAIQVLEGLTAVRKRPGMYIGSTSTSGLHHLVWEIVDNAVDEHLAKHGSKIEVTLHKDGS
VTVQDNGRGIPTGMHKTGIPTPQVVFTILHAGGKFGGGGYKKSGGLHGVGASVTNALSEWLEVTIFRDGKIHKQRFEYWI
DEKGVEHVGEPRTGLEVTGTTNKTGTRVTFKPDRRVFQNGIAISYDTLAERLQEIAFLNSGLEVILKDERTDQVETFKYD
GGASEFVKFLNEGKSVMHDVIHFASEKDDIEVEVALQYNDGYTETIASFVNSIPTRGGGTHETGFKIAYTRAMNEYARKT
AILKEKEKNLDGTDLREGMMAVINIKMSEVEFVGQTKDQLGSASARSAVDAIVSEKMAIFLEENPQIAQSMLKKAIQSAK
AREAARKAREEVRSGKKGRSESSNLGGKLTPAQSKDYARNELFIVEGDSAGGSAKQGRDSKHQAILPLKGKPLNPEKAKL
ADILKNDEYRAIIATIGAGVGSEFAIEESNYGKIIIMTDADTDGAHIQVLLLTFFYRYMKPLIDNGKVYIAQPPLFKVTK
KAGKNSAVHYAWTEDQLQALHKELGKGVELQRYKGLGEMNPNQLWETTMDPGSRTLLQVQIEDAAKAERRVSTLMGDKVD
PRKRWIIENVDFTESQE
>lcl|NZ_CP026520.1_prot_WP_042228320.1_2383 [gene=gyrA] [locus_tag=PC41400_RS12195] [protein=DNA gyrase subunit A] [protein_id=WP_042228320.1] [location=2638360..2640822] [gbkey=CDS]
MSSLEQFLPAFLEEVVGDRFGRYSKYIIQDRAIPDVRDGLKPVQRRILYAMYDSGNTPDKPYRKSAKTVGDVMGNYHPHG
DSPIYEGMVRMAQPWKMGHVLVDGHGNWGSIDDDPAAAMRYTEARLSPIAMELLRDIEKRTVLFKDNFDNTTKEPVVLPS
RYPNLLVNGVSGISSGFATEIPPHNLREVIDACTALIDAPEIGLEELMRHMKGPDFPTGGIIMGEEGIRDAYNTGKGRIY
VRSKTEIQDLKAGRQQIVVTEIPYQVVKSRLVTAIENIRLEKKVEGIAEVRDESGRDGMRIVVELKKEADAQGILAYLLK
KTDLQVTYNFNMVAIVNKTPKQLGLKDMLSAYIEHQKEVVTFRSQYDLERAEDRAHVLEGLVKALNILDEVIAAIKASKN
RLDAQKNLVTQFGFTERQADAILVLQLYRLTNLEITSLEKELKDVQKTINHLRGILGSTKKLLGVIKDEMTEIRTKYGME
RRSDIQNEVEEIKVNLEVMVTPEDVLVTLSNEGYIKRTSMLSYTRSGGAVDNTGLKEGDYLRHLKEVNTIESLLIFTNKG
QYFLLPVHLIPEYKWKDNGTAIVNIIPIAKDDRIVDVIPVKDFNEEGKSLVFITRKGQVKRTDLKEYYTNRSSGIVACKV
GVGDEILHVQLSDGTKDIMLATKQGMSIRFREEEVNPMGRAAAGVRGLQLREGDEVVAADWVYGDEGEVLVISDLGYAKR
TLLVDYPLQGRGGKGICTFEFKEGKRVKSNGSELVGMFYFKQDYEITAWNSSGEKTVFKTNQAPLDDRKSIGKQLVPVSK
GGSIAALVKNASLDKRPPNA
>lcl|NZ_CP026520.1_prot_WP_009675479.1_2384 [locus_tag=PC41400_RS29755] [protein=hypothetical protein] [protein_id=WP_009675479.1] [location=complement(2640840..2641007)] [gbkey=CDS]
MHIKKSALGQRALDLGLLKDPGWLERLDEQVPLWVVLEIAFQLMETMDPPTTSYD
>lcl|NZ_CP026520.1_prot_WP_042228319.1_2385 [locus_tag=PC41400_RS12200] [protein=hypothetical protein] [protein_id=WP_042228319.1] [location=2641105..2641350] [gbkey=CDS]
MQRVEIGKGCGLKGKRKRIAASVIDLIWSESDRAYQIAQKQVEGIWCETLGEHKRRDGLKTRELNRVKDAGPEEQKQLWG
K
>lcl|NZ_CP026520.1_prot_WP_009675620.1_2386 [locus_tag=PC41400_RS12205] [protein=MarR family transcriptional regulator] [protein_id=WP_009675620.1] [location=2641565..2641969] [gbkey=CDS]
MYSVEFAKLWTKLSKELKSNMEDQLAPTITEGQLHVLELLAANGTMKPSDFIEYLATTPAAVTTLLDRMERAGLITRSRD
EKDRRVVWVDATDKGKSERERGLQIREEVLDAYLSRISSHNQQLLVYLLGKVAN
>lcl|NZ_CP026520.1_prot_WP_042228317.1_2387 [locus_tag=PC41400_RS12210] [protein=RluA family pseudouridine synthase] [protein_id=WP_042228317.1] [location=complement(2641934..2642671)] [gbkey=CDS]
MAGPKHAADGAAIPVLYEDNHVLVVEKPVNMPTQEDSSRDPDLLSVLKEDLKRRHHKPGNVFLGLVHRLDRPVGGVMVFA
KTSKAASRLSDSVRTRTFDKTYAAVVRGRPKEPQGSLRHWLLKDTRTNTVSVVKPGKDGAKEALLDYRVLGSAEGLSLVL
VKLHTGRPHQIRVQLAAIGCPLYGDQRYGAGANEPGQQLALWSARLSFPHPTQRETLTFRSSPPERHPWFLWQELVRNLP
EQINE
>lcl|NZ_CP026520.1_prot_WP_042228315.1_2388 [locus_tag=PC41400_RS12215] [protein=class I SAM-dependent methyltransferase] [protein_id=WP_042228315.1] [location=complement(2642675..2643541)] [gbkey=CDS]
MYVANRWNDYELLDTGGGEKLERWGSYVLRRPDPQIIWPLTAETANWKQAEGHYRRSSSGGGNWDFRQNLPERWTISYDN
KLRFHIKPTGFKHTGLFPEQAANWDWMMDKISNAGRPIKVLNLFAYTGGATVACAHAGASVVHVDASKGVVQWAKENLEL
SGLADRHVRFITDDVFKFVQREQRRGNKYDAIIMDPPSYGRGPNGETWKLEDDLYPFIETCMTILSDNPLFLLVNSYTTG
ISSTVLKNILTLAMHKRYGGHITAGEIGLPITASELTLPCGILGRWEA
>lcl|NZ_CP026520.1_prot_WP_042228887.1_2389 [gene=glgB] [locus_tag=PC41400_RS12220] [protein=1,4-alpha-glucan branching protein GlgB] [protein_id=WP_042228887.1] [location=2643793..2645742] [gbkey=CDS]
MAGKQIPGSEDIYLFHEGTSYHSYRLFGAHAEKRGKTTAVRFTVWAPHAREVRVVGDFNEWNGENHRMETVSESGGIWTL
AVPGLAEGSLYKYEIHTPDGRIILKADPYAVYSELRPETASRFYNLDGYKWGDQNWRRRRARKPVYDQPLNIYEVHLGTW
KKKSREEDELLTYRELADELVDYVLEMGYTHIELLPLAEHPFDRSWGYQATGYYSVTSRFGEPKDFMYFVDRCHQAGIGV
IMDWVPGHFCKDAHGLRQFDGQPLYEPDQTHLAEKPEWGTLAFDFNKPEVHSFLISNAMFWMDKYHIDGIRVDAVASMLY
LNFGKPQSMWVTNEHGGDENPAAIGFLKKLNTAVFGSYPNALMMAEDSTDWPQVTSPVDAGGLGFNYKWNMGWMNDMLKY
MKMDPIHRKHHHNLITFSFMYAFNENFVLPLSHDEVVHGKRSILHKMPGDYWQKFANTRVLYGYMSGHPGKKLLFMGGEF
AQFDEWKDLEDLDWFLLDYELHGKMQTYVRALNRFYQEQPALWQNDHGHDGFRWISPDDSSQSIVAFIRKGKKPKDDLIV
VCNFTPVVHPDYRIGVPEPGTYAEIFNSDDEAYGGSGQANRQSLESEAVPLHGFDYSLSLHVPPLAAVYLKRSQVERTSR
RKGKSKDAK
>lcl|NZ_CP026520.1_prot_WP_042228313.1_2390 [locus_tag=PC41400_RS12225] [protein=glucose-1-phosphate adenylyltransferase] [protein_id=WP_042228313.1] [location=2645732..2646880] [gbkey=CDS]
MRSKECIAMLLAGGEGRRLGVLTQQLAKPAVHFGGKYRIIDFTLSNCTNSGIDTVGVLTQYQPLVLNSYIGIGGAWDLDR
KNGGVTVLPPFMEQGGGDWYQGTANAIYRNIGFIDRYEPEYVLIISGDHIYKMDYDRLLTYHKTKGADATIAVIEVKWEE
ASRFGLLSVNEESEITEFAEKPKEPKSNLASMGVYMFTWTALKKYLIQDNENPDSSKDFGKDIIPSMLEEGANLVAYPFQ
GYWKDVGTIQSLWEANMDLLDDPTQLNLNDKSWRIYSVTPYQPAQFVSNGAKVRRSLVNEGCSVFGDVDHSVLFYGVHIG
EGSVIRDSVIMPGASIGSNVTIRKAIIGENTIVDDGAVVGDGEEIVLIGGGEHIRATTESKG
>lcl|NZ_CP026520.1_prot_WP_042228312.1_2391 [gene=glgD] [locus_tag=PC41400_RS12230] [protein=glucose-1-phosphate adenylyltransferase subunit GlgD] [protein_id=WP_042228312.1] [location=2646888..2647997] [gbkey=CDS]
MKHVMGVINLVNEPDDLEELTLHRCAGSVPFGGRYRLIDFILSNMVNSSIGNVAVFTQHKYRSLMDHLGSGKEWDLDRKR
GGLFVLPSVQDDPAGMSRGDLFQFYSHRDYFYRGREELVIIGRSHMVCNINFEDAVRYHEDMRADVTVIYKEAEEEYSGS
YRRLAVRPDGRVTVIEDQTGRLRTDYVSMEMYIMSKALLLDMVETCLAQGYDHLVRDGIMKNVDKLNVVGYHHDGYLGIV
NSIQGYYRHSMELLDPSVLRELFFRKDLIYTKVKDEPPARYEANASVRNSLIANGCVIEGKVDNSILSRGVKIGKGAVVK
NSIILQNCSIEENVVIENAILDKDVFISRGRVLTGDVRAPFIAAKMKVI
>lcl|NZ_CP026520.1_prot_PC41400_RS30270_2392 [gene=glgA] [locus_tag=PC41400_RS30270] [protein=glycogen synthase GlgA] [pseudo=true] [partial=3'] [location=2648109..>2649482] [gbkey=CDS]
MNILFAASEAAPFAKTGGLGDVIGSLPKELKRNGADVRVILPKYDEIPQVFKDKMETANLVTFRVNLGWRLQYCGIQTLE
HEGVTFYFVDNEFYFRRSGHYGYGDDAERFAFYCRAVIDSLPHIGFKPDVIHCHDWQAGMIPVLLRAQYSPYYPFYEDIK
TVITIHNLRYQGLFSKETMMDLFGLDAGYFNGDQLEMYDGGSFLKGGLLYSDRITAVSPTYAEEIRTPFFGEHMDSLLRY
RSEATVGIVNGIDYDLFNPVTDPHLAVTYRDSLSKKRLNKTHLQQRLGLPVRDDVPVISLVTRLVQQKGIDLIRHVLPEL
LSLDAQWVIVGTGDHEYEQLFREAAAQHPDKLAAYIAFDDGLARQVYAGSDLFLMPSLFEPCGIGQLIAMRYRTVPIVRE
TGGLKDTVHPYNEFTGEGTGFSFSSYNAHDMLHTVRRAVHFYLHDQEAWRNLAANIKK
>lcl|NZ_CP026520.1_prot_WP_241688894.1_2393 [locus_tag=PC41400_RS30275] [protein=alpha-glucosidase] [protein_id=WP_241688894.1] [location=2649789..2651444] [gbkey=CDS]
MYPRSFKDSNGDGIGDIPGIISKLDYLKELGVDVIWLSPVYQSPNDDNGYDISDYRGIMDDFGTMADWEKLLAEVHGRGM
KLIMDLVVNHSSDEHAWFLESRSSKENPYRDYYIWRPSRDGQEPNNWTSIFSGPAWEYDGQTGEYYLHLFSRKQPDLNWE
NPKVREEIYDMMTWWLDKGIDGFRMDVINMISKTEGLPSAGGDRYAWGGDYFMNGPRVHEFLREMNEKVLSRYDIMTVGE
TPGVTPDDAARYANADGTELNMVFQFEHMELDSGWGGKWDPCGWKLKDLKATIGRWQTGLHGRAWNSLYLNNHDQPRMVS
RFGNDREYRVESAKMLATFLHTLEGTPYIYQGEELGMTNVPFKGIEDCRDVETLNMYRERVTEGGGNHDEVMNAIRIKGR
DNARTPMQWSTEANAGFTTGTPWLHVNPNYTEINAEAALADTDSIFHYYRKLIALRKTHPVIVYGSYEIVSEEDERIYAY
TRTLGNDRLFVMLNFTGETVSFASPYPLLAGGGGELLIANYPVGEGSETSGLDAELQLRPFEARVYLLKAV
>lcl|NZ_CP026520.1_prot_WP_042228308.1_2394 [locus_tag=PC41400_RS12240] [protein=hypothetical protein] [protein_id=WP_042228308.1] [location=2651719..2652987] [gbkey=CDS]
MKKWLLSGAAFLLLASSASILAPVRAKAMVINVPWDVTLLRETMMYEDEAFTKPAGSLSPLQTVHITKRDSGWSMADHTE
EKGYLIETWQGEKWIKADPETLVPGKYKAETATITLVDKQELYDEPLQETASGTIISPQQVTVTGTIEWDGADFGPPRTI
TFYRIQTWLGDKWIYDSFYKEAVREKAVSTTVDLFQDTDSFELPYTGLTKKGTLKTGRYPVTAVWELPPTLGDGSPTWYK
VSLPEGERWILPGDELAGVRETTGKIELPTGGKGFAKPTYYDDENYTEVAPGSYDAFETWKDGELEWVHIRTESEALWVN
PGLALKVRPAGIIPTDDDIPLTSRSATYKYPDWSRIHNMKGFFASQTVKAFEKWLSPEGITWYHVHGTWDDWWIPEKDAE
EPAAQPVEPEMNSAAVFGQNGQ
>lcl|NZ_CP026520.1_prot_WP_042228306.1_2395 [locus_tag=PC41400_RS12245] [protein=NUDIX hydrolase] [protein_id=WP_042228306.1] [location=2653105..2653722] [gbkey=CDS]
MSAKWLEWAKQIQAISQIGLAYSKDVYDLERYEMLRDLSLEIMSEYTDLHREKIRDLFANETGYATPKVDIRAVVFREGK
LVLVREKLDGYWALPGGWADIGLSPSEVAVKEVKEEAGLEVKADRLLAVFDKKFHDHPESPYHVYKFFVLCTVTGGEIQG
GLETSEAAFFGKEELPELSAERNTKAQLLTMFEMCEDPGRPVLLD
>lcl|NZ_CP026520.1_prot_WP_042228304.1_2396 [locus_tag=PC41400_RS12250] [protein=DUF4362 domain-containing protein] [protein_id=WP_042228304.1] [location=2653819..2654271] [gbkey=CDS]
MNKKLAIVLMGAAFLAGFMPFVAVKTKWFQPAAAQANADILVNLDQRDLKRLDELVGRFNRAQGDNLMVISPTIDSGPWI
HDVYTDGTVIVWTVDNARDAYSSPKQKQTYTCASVEKIETSERYVLQLSKCGGGQNDKLPILDIEKNRER
>lcl|NZ_CP026520.1_prot_WP_129112475.1_2397 [locus_tag=PC41400_RS12255] [protein=discoidin domain-containing protein] [protein_id=WP_129112475.1] [location=complement(2654541..2657291)] [gbkey=CDS]
MKAFSVKSKKLLVVLTAALVLPQPLLMPPGARAQEQRAVQTAMAAADQAADRTAWASSEVSGREASKAIDGDAATYWQPS
SADREDMNVWLAVDLGQNTPVNKAVFSLNRADNLASYAIQYSSDGATWSTAYSRNQSLSKDESAFFEEVTARYFRLNLTL
GKNLNVNVFEFQLTKDVNSENPVPSDLKRIYFANDDGQEVPLNGELTLQKNAVKTLNVKGVTSSGSIVSLPGTAVSWISA
AKSIEISSAGAVKALKTGASMVNAEATVNGRKIKSPDFWIVVEDPAEFKGEAFIAGTSLSHPTLRTEIGQPAILKAGDAL
PAIAVRANTALTLSGEVRRNGNPAPGITLPVTAFDKGQRKTVQLAGPSTAPGLYEIRLTFSSPGRPDYHDTFSFTVPDKA
KAAGTQSSIAYLNEDDRMTYVPDFKGNRILDFSGSGYKGGGVPIPNVQAKAVVSPGAGDDTRRIQDAIDSVSQMPLGPDG
FRGAVLLTKGQFEVSGTLKITAGGVVLRGEGQDAGGTVILGTGTTARNLVEIGGTSGVVTDSASKTAITDLYVPSGASSF
HVADASAYKAGDKVIVRRIGNDRFIHEIGMDHIYMRPGQGGTQQWGAFNLDFDRTILKVEGSRITLDVPLANSIDLKWGG
GELYKYSDAGRIEQVGVENMRVDSEFDKSVIDTVMDNEKTDPYYADEKHAERFVVFNSVKNGWLREVTTTHLSYSTALVS
RQAKWITVQDTTTLDMVSIITGGRRYAIHFQGQQSLAQRNHVETSRHAFVVDSRVQGPNVFLDGTTDKNFNTSEPHHKWS
VGGLYDNIQAPISIRDRAWLGSGHGWAGANYVTWNTEGALTSQQPPTAQNYAIGHVGTVVPGLVPSTYDPRPRSEAFWES
TGALDAGQPVQAAASGEKRPACAAKYFPASARVRFA
>lcl|NZ_CP026520.1_prot_WP_042228303.1_2398 [locus_tag=PC41400_RS12260] [protein=DUF4962 domain-containing protein] [protein_id=WP_042228303.1] [location=2657542..2659845] [gbkey=CDS]
MSKSLYQPRSGVLTVQYGPDAQTKVIENPPRFTWIPTRLEKDCYVLEISERDDFPSGLTRTYAPVPYNFFTPDKTFAPGT
YYWRYALYENDKRTSEWSVARSFSVPENLPETPLPAKGERFKESGYARPRLWLQGEEGIARYREKIAADPEGTGWKLFYE
LSVLPWAEREPHAEPEPYPGNKRTVELWRQSYIICQEMLYAIRHLSIAGKVLDDSRLLGRAKDWLLAAASWDPAGPTGRD
YNDEAAFRMAGALAWGYDWLHGELDAEEKERVKRSLAARTRQVADHAIRRSKIHQVPYDSHAVRSLSSVLTPCCISLLHE
EPEAEEWLHYTLEYFWTLYTPWGGEDGGWAEGPHYWTTGMAALQEAIEWIRGCTKLNLFDRPFFHKTGDFPLYVYNPNAS
TRTTFGDNSSLGGKFGAKTGYNMRHLAGVTGNGRYQWYYERIKEMNPEAAKLFHNHGWWDFRFDELVYRQAYPQIRPEAP
SDAEELKWFRDVGWVAMHHRMDDPGEQIMLLTKASKYGSVSHSHADQGAFYIHAFGEALAVETGYYVAFNSTMHRNWRRQ
TVSKNSLLIDGAGQYADTDKSLNMAASGTVAEAKRDKIGSYVKMDVTAAYLEQVPYAKKVEREIYFVDNSYFVLVDSVDL
NRPGKIEWLYHALYEMDLREQSFRVNGKKASLEGRFVFCSSGELTLTQSGEFAGVDEKEIEGLARQWHLKAESPPAASHR
LVTLLVPKKPDEEDRYVSYFMDDQGFVVSLYFTDNGKTRKIDVQKAF
>lcl|NZ_CP026520.1_prot_WP_042228301.1_2399 [locus_tag=PC41400_RS12265] [protein=cupin domain-containing protein] [protein_id=WP_042228301.1] [location=2659943..2660260] [gbkey=CDS]
MSNINVWEDAEPGVKRKIMQPGQSLMLMEVHFEPGAEGYEHAHPHEQLSYCMKGRMEFTIDGVKKVIESGESICIPPHAR
HGAKALEASILLDCFTPVREDLVKR
>lcl|NZ_CP026520.1_prot_WP_042228300.1_2400 [locus_tag=PC41400_RS12270] [protein=alpha-L-fucosidase] [protein_id=WP_042228300.1] [location=2660301..2662142] [gbkey=CDS]
MRENDPLMDGAEAEARRSRQPGNPGGEMRASGDGSSDARLAWWREARFGMFVHWGPYSLLGGRWKGEEVQAAYAEHIQLR
GRIPVREYEKIAAAMDMSGFHADEWVAAAKQAGMKYMIVTAKHHDGYAMYDSQACGYNIVRLGPSGRDPIRELSEACGRG
GITFCLYYSHAMDWHHPDSQGNTLDYPDNIGAWDSLESWTGDETKSRRYDRYLREKAFPQIRELLTNYGPVGILWFDCGH
KITDEQGREFAELVRSIQPGCLINKRIRSDEFADYGNPGDNQLSVRVTRPDWESIFTLNDSWGFRHADRNWKTPEELIRI
LVETASRGGNTVFNVGPEGSGRIDAHSRRLLEEAGRWMAVHHASIRGTEGSPIGRPEWGCCTANRQERKLFLHILRRPPG
GRLIVSGIASPVTAIRWLHDPAAPAPRFRRLNGLDLLVELPGPLPDNASSVLVLQYEGELAADPHKRLMEEGVTTFPVFD
GEAAGERLRYDTGKKGRDNVTDWSRAGDSLSWTFRAAAPGRYAVCLSYAAEPGVSAGGSYSVCCGDHALRAEVVPTEGPY
DFTVREIGGLEIPGPGTYTLTVRAEDIRGGYLMNLRQVVMKALTEEEKPDVQF
>lcl|NZ_CP026520.1_prot_WP_042228298.1_2401 [locus_tag=PC41400_RS12275] [protein=glycoside hydrolase family protein] [protein_id=WP_042228298.1] [location=2662129..2663151] [gbkey=CDS]
MFSFKPAPVGGGFRMEGYWVWCGSAVKGEDGRYHLFASRWPDSLPMHPGWLVSSEVVRAVSDTPEGPYTYAETVLPARGA
EYWDGRSTHNPHIVRIGDTYVLYYMGSTHPFADPEPGEPLRPEDPVTVVARSNKRIGIATASRPEGPWLRRDEPVLTVRP
GRFDSFLTSNPAPCVMDDGSVLLVYKARKYEGFRHGEMTIGAAKAPHFEGPYTRLAEGPLFPPDFNIEDPFVWKTEDGFS
MIAKDMTGRLCGEKYGGIRAYSPDGVTWSLCDPPKAYSRTFLWDDGVTRQAGNLERPFLLMEGGVPTYLFAAASDGTKGF
HDCTKTWNMVIPLKEGPPTC
>lcl|NZ_CP026520.1_prot_WP_042228295.1_2402 [locus_tag=PC41400_RS12280] [protein=SGNH/GDSL hydrolase family protein] [protein_id=WP_042228295.1] [location=2663145..2663780] [gbkey=CDS]
MLIRKNDVVLFQGDSITDAKREPGHELGRGYARMAAAQFGRMYPELGVTFLNRGISGNRVRDLQARWEADCLELKPGVVS
IYIGVNDTWRRYDRSDPTSAEAFAAGYRDILDRTVEGLPGVKLMLVEPFVLPVPEDRRHWREDLDPKIHGVRDLAREYGA
AYVPLDGLFAQASVKAPCPYWAPDGVHPSPAGHALIADAWLKAAGVSRALF
>lcl|NZ_CP026520.1_prot_WP_042228293.1_2403 [locus_tag=PC41400_RS12285] [protein=ABC transporter permease subunit] [protein_id=WP_042228293.1] [location=2663841..2664791] [gbkey=CDS]
MASKIQPSDKVRYVQRHEVGSNLRKYKALYLLSIPGILYFIVFKYVPLMGSVMAFQNYNLFKGITGSPWVGLDNFIRMFE
YDEFLEIFTNTLLIGLYDIVIAFPVPIILALLLNEVRKMVYKRILQTVVYMPHFLSWVIIGGIAVGILSPTTGIVNTMLG
WFGMEPVYFLGENSYIRSILIGSGVWKDAGWGTIIYLAALAGVNPELYEAAKIDGASRWKQTTAITIPTIMPTIAIMFLL
HIGHFLDFGFERVYVFLNSLNSESGEILDTYIYRVGLLGKQYSYTTAIGLFKSLIGLVLIMIGNRLSKKATGESLY
>lcl|NZ_CP026520.1_prot_WP_042228291.1_2404 [locus_tag=PC41400_RS12290] [protein=carbohydrate ABC transporter permease] [protein_id=WP_042228291.1] [location=2664808..2665680] [gbkey=CDS]
MVKESKGERSFQGFNYVLLTVVAICMILPIVHVLAKSFSSELALNQGRVSVFPVEFTLSNFAFVLSDSSVWQGFLVSVFV
TVVGTLINLIATTTLAYPLSRPEYIGRRYLLLMVLVTFIFSAPLIPNYLLIKNLGLLDTVWSLIIPGMISAYNLFIMRSF
FSGLPGELVDSGRIDGCGEMRLIWSIVLPLSKPVMATMGLFYAVAHWNSYSNALYYINNRNLFPLQIRLREIVITDNFGE
MTSTFENTLNISPEGIKMAVIVVATVPILLIYPFLQKYFIKGMLIGSIKS
>lcl|NZ_CP026520.1_prot_WP_042228290.1_2405 [locus_tag=PC41400_RS12295] [protein=ABC transporter substrate-binding protein] [protein_id=WP_042228290.1] [location=2665794..2667344] [gbkey=CDS]
MKKGWGVLLAAVLTTGTLAGCNSKEPAKESAAGAGPVKFSVSFSTGGNAYLESASDINKDKWVLELEKRTNTDLDLKPLS
HKEFDQKMSLMFAGNDIPDVVSNLRGGPTTSSMSGSVEAGVFMPLDDLLKENAPNLMKLVPKEAWEETSYDGKIYGIPGW
LTNPSRRATFIRTDLLEKANLKQPVTVEEYLDVLRAFKKMGVENPYQIREQFKYADTFLGAYDVMPYQFELMNGEVVPKF
FDAENMMKALQTYKTMYEEGLIPKDFATVSQTDYNKNIAAGKSGIWSANGEGLLSFRGKIKDAVKDGKVDIVASPKGPEG
KGGHGVYGYISTAKYINSKVDKEKAKKIIQFFDWMSTEEAEEFFTFGIEGDTYTKENGKINYKQPKTKEEIDEANYRQTT
LWMVHDLTYNKKLSELTEDGKDMVKAFDNVLSKEGVGGITFTTSLASFTKYPDLASGGDTGPKFILDQMIQMIYGKRPIS
DWPKVIEEYKAKGGKEIIKEATERYNKKEGVVDRAR
>lcl|NZ_CP026520.1_prot_WP_042228287.1_2406 [locus_tag=PC41400_RS12300] [protein=response regulator] [protein_id=WP_042228287.1] [location=2667480..2668499] [gbkey=CDS]
MHLLIVDDEPIIRKGLVKMAELYPFEFAGIHTAASGTAALEIIREHEIDLVFTDIRMPKMDGLELCRIIHEQYSHIPCIV
ISGFGDFSYAQKCLFYGVKHYLLKPVTAPDLHEVLADFMKNRVKGFVALSDCVDVIEQMEQLVWNLEEDRLRTLIGEWKQ
NCMSSGMNLLQLRQLIKECMALLLKRLQARGLPVKNEWEDKRCVSKDQIFEEMQTRLGDIMEKLLQQRSGTYKNLLDEVK
TYIDTHLAEEITLDEVAEMAGFTPTYFSSMFKKMTNETFVKYRIKKRMEKAQQLLAIPHIRIVDVAAKVGYEDYPHFTKM
FKKTTGYSPSEYRQQLGIK
>lcl|NZ_CP026520.1_prot_WP_042228285.1_2407 [locus_tag=PC41400_RS12305] [protein=sensor histidine kinase] [protein_id=WP_042228285.1] [location=2668496..2670424] [gbkey=CDS]
MIKWNLSRKIFLYLLSIILITLTSVGIFTYKKASDELDKQNRLHMNQILTNALHHTDVYVKSYDRLTISLLTNRDIKKFV
DLPPVPEPYDYYKFWSTIRETALQPIFIRNPEIVSMYAISFAGNGIYGYSNDQAEGTYPREQMLGQLERLKNVTEPDGTL
SITTDSIVPAMNGNVIRLSRQIKGLQSAEYKGILAVEIRSNDLSMLWKGIDLGEGGYFFIVNDKGEMIYQPSKDGVGPQA
LPELLPRIVHSGETASFEHKGPGGEERLYMSRKSSYSGWNLVVSMPLSELKKPVTAIRTTTILIGLFTLAIAVALAYRFG
RSITRPILVLIQGMRQTEQGKWSTVPLPNRRDEITELMQRYNLMVGRLSELVEQVYEGELKQQKTQLERQQAEFQALQLQ
INPHFLYNTLETIVCYAVVQDSEEISEIVKSMAYMLRYSVQTNLEEITVANELNHVLNYMIILKHRHDREFELDVAVPPQ
FLLKKMVRLTLQPLIENVFQHAFPDGLEDHHWIRIDAGEEGGRFWISVEDNGTGMTRETLENLQSKLKTQQLTDASAAAG
LEESGRGGIGVLNVHRRIQLVFGAEFGLRVESEPGIGTKMTMIMPVSKPANGSPDEGRGELPEPRNAWRPEPGDKEGTGR
TE
>lcl|NZ_CP026520.1_prot_WP_042228284.1_2408 [locus_tag=PC41400_RS12310] [protein=glycoside hydrolase family 95 protein] [protein_id=WP_042228284.1] [location=2670654..2673104] [gbkey=CDS]
MTANKLWYTSPAQAWTEALPVGNGRLGAMIYGRVEEEVISLNEESLWYGGPKDRTNPEAAAALPEVRRLLMEGRVTEAQE
LAHMGLTPIPKYAGPYQPLGDLRIWFAEHEPDAGTYRRELDLATGLCRVEYATQGASCTRELLASAPAGVLACRLTTAHP
EGLNFRFHLGRRPFDEGAAPAGPDAVLMQGRCGPDGVRYAALASVSAEGGTVRTIGDFVHVAGAAEATIYVAAQTSFRHG
DPAAVCRRQVEEARRKGYEAVKAEHRADYMPLFARMSLELGTPDADIRLLPTDERLDRVREGGEDPELLALFFQYGRYLL
LASSRPGTLPANLQGIWNNDYQPPWECNYTLNINLQMNYWPAEVCNLSECHEPLFEFIDRLVASGRETARKLYGCRGFVV
HHNSNLWAESGINGMLPRAAVWPMGGVWLALHLWEHYRFGGDSHFLAHRAYPVMKEAAQFLLDYMTEDGKGGLLTGPSVS
PENKYVLPGGKIGYLCMAPAMDIQLTRTLFGAVREAADVLARERGARGADRPDPCSMDGGAGSPTVSAEAGLAGTEARSG
GHAAACGDAAPPAEAEWLERLTAAESRLPQPATGSRGQLLEWLGDEEESDPGHRHISHLFGLFPGELISPVRTPALAEAA
RVTLERRLAGGSGHTGWSRVWIAHYWARLRNGDEAHRHLTALFRHAADPNLFTEHPPFQIDGNLGGTSAAAEMLLQSQEG
MLDLLPALPSAWPCGRVKGLRARGGYEADLEWERGLLTAGRVAASVAGILRIGYKLPFTACLDGKKAAGESRFSESCGLY
VTDIEHEAHAVVEIRL
>lcl|NZ_CP026520.1_prot_WP_129112476.1_2409 [locus_tag=PC41400_RS12315] [protein=hypothetical protein] [protein_id=WP_129112476.1] [location=2673569..2674750] [gbkey=CDS]
MGKGKRSVFLLLAVWGVLFRPEAAYMHGFSAAGDGGSVVLSVQTTPDAAYRSETGVTLSVGKLFGQSVRRKAESFPYTDT
FMKVRETGGEHIYPVSAMGELYDSSGHEVIRPGEALRGQLVKQIAVLHRKHYGELISWNDAQKAVPRQAKFEVLDLETGL
AFRVQRRAGRQHADVQPLTKKDTAVMKSIYGGKWSWNRRAVLVRYDGHSYAASMNGMPHGGDGIPENGFSGHSCIHFAGS
TTHKSRSKDTAHQFMAHRAAGMAGSFAGKASPQELADLVVTALDQHDDAMLELLGRPHNRDRLEELKKDSRSFVSVRRLT
EWKPDEARKAEQEYVFELPVELAFEEAGKRPRKAICRFAFTRASLYDPWELTQFEYPEPDRPGKRTEGGRNSR
>lcl|NZ_CP026520.1_prot_WP_042228282.1_2410 [locus_tag=PC41400_RS12320] [protein=L,D-transpeptidase family protein] [protein_id=WP_042228282.1] [location=2674755..2675507] [gbkey=CDS]
MNAGAGINMNKGQTLLKIKGVALTAVCLLNVLLPGNAAAERITLPADVTEPAIPIVADRGPFSIEVYPLRHKLVVFKDKK
HYKTYPVAVGNLATPTPVGEYQVIYKGKNWGPSFGPRWLGLNVPWGAYGIHGTNKPHSIGQHLSHGCIRMRNRDVEELYD
LIPIGTKVTIAGHVLGDPGREPRKLAQGDIGADVQLIQNRLKSAGYYSGVCNGKFLASTAEALRRFQRDNGLRADGVVSI
SVYEKFGLME
>lcl|NZ_CP026520.1_prot_WP_042228280.1_2411 [locus_tag=PC41400_RS12325] [protein=MHYT domain-containing protein] [protein_id=WP_042228280.1] [location=complement(2675582..2677477)] [gbkey=CDS]
MQLEAGNYDALPIILSILMSMTTSCTALDMADRLLRGDRSFRKMLLTAVVTGLGVWSMHFLGMLALKLPFEVHYHLPLLA
VSLMLPILAFFLSLLILAPDRKPGKRFFTAVGLLAGLAFVSMHFSGMLAMKIPAVYHQTAGSLIVSIALALLASYLMLAL
AIYARGLPAYNLLSPWKTGGSLLVGGLIAGMHYTAMAGTSFTPSDRELSLTSSSIIGNTLLTVLVGCTTFCVLFWVVLVL
YRDRQNVLTASRFNEQRYTALFEHSPDLVICMDPVKKMIVSANPSVYANTGYTQAELMLLSLHDLLYSREEKLKVMTALA
EAARGKPSKIEVSIRTKQGARKYHRTSVFPLEHNRHLLVYFVSKDITDRIVVEQELRKAKDLAESAVKTKSEFLDTMSHE
IRTPLTGIIGINQLLSESAESPDDQELLALQAKSSQALLRVINDILELSKMEAGRISLHQEPFQLRELIRECLDFFAIII
RNKELAMDTSIDPAVPETLVGDYGRVRQILVNLIGNAVKFTDQGLISLEVRLSERGDEGCLLEFAVTDTGIGIDCSKLDR
LFQPFTQLDGASNRKYEGTGLGLAICRKLVTLMQGRIWAVPGRSVGTEFRFTIRLRELRHFMDEGDSQEAG
>lcl|NZ_CP026520.1_prot_WP_042228278.1_2412 [locus_tag=PC41400_RS12330] [protein=HAMP domain-containing sensor histidine kinase] [protein_id=WP_042228278.1] [location=2677759..2679270] [gbkey=CDS]
MSIRARLLASFTLLLVATVLLFTAAAFIVTTAVTGDVRGISSFYNVHLRLKPLTTEQETLFLDLKHLAKTKSEELLVPDI
YKEYDFQLKVAKASLLIRKGSGSAFISPALAENQPEAVLPPYEAENVHMRSTLQTGKNFFGYAKYDFRFADGEQGRLFVF
REVSPFAVLMSKLLPLFVSVLFLLLVGSNIYIYLFLTRRMIRPLNLLRASAARIQRGDLRFRIEPGEDEIGQVSAAFEEM
RGKLQESEELQKQYESNRKELIANISHDLKTPLSAIRGYVEGIRDGVADTPDKLESYVDIIYSKAEEMDHLIEELFLYSR
LDLNKVPFALQEVDLTRFVADVVYELRFELEKEGVGLETGGHSAGPLHVRIDPMHMKRVVNNVVGNSLKFMEGPHKTIAF
DLRRDGAAAVLEITDNGPGIDPAALTHVFDRFYRAETSRNRESGGQGLGLAIAKQIVDNHGGSICAENASESASTRHPQT
GTRIVIRLPLLKGNHTDEKNSDH
>lcl|NZ_CP026520.1_prot_WP_042228277.1_2413 [locus_tag=PC41400_RS12335] [protein=response regulator transcription factor] [protein_id=WP_042228277.1] [location=2679248..2679937] [gbkey=CDS]
MKKILIIEDEATIAQLHRDYLNLHGFQVEISTRGDDGLNKALTEGFDLIVLDLMLPGKDGFEVCRQIRMAMEVPILIVSA
RKDEFDKIRGLGMGADDYMTKPFSPGELVARVKAHLTRYERLVGLKETAPGEIRIRDLAIEPESRRVYIRGKEVPLPARE
FDLLLFLARHPNRVFRKEEIFEKIWGMDSMGDIATVTVHVRRIREKIEADPSAPEYIETIWGVGYRFSG
>lcl|NZ_CP026520.1_prot_WP_042228275.1_2414 [locus_tag=PC41400_RS12340] [protein=hypothetical protein] [protein_id=WP_042228275.1] [location=2680045..2681331] [gbkey=CDS]
MKKTLLQLLTVSAVAASVMLPLNAAPAAAQSAKTPAVSANGDLSPAGGHHSDHGRLMLTQQELVSRIAQLTEKKPLVIQQ
WLKNYEPGARVTKGEAARLGFKAIAHIPSEHAAVTDIKVLSVATLEITFDSPLTAADVNLDKAKANFQFDNGMTIRNVPQ
LKNGARSTYVVPVTVQQPGKTYSLTYKGKAAGKFTGSAHTVKLAVARQVTADTFELESFRKDGVVDYGYVVESYFGTRGG
LEMYLNDRNASGGKSYDIISSLRDMSVTLTPEGGQPIRAAYVPYTQATDDRQAPKFRLPAGQTLTPGVTYKVTADWATVT
EASFTARSAAPLVIASAKAVSPASVDITLAADPRDELFSGRSVELTDGDGKKLTAVYKVTSRKGAVGTFELAAGTQLSPG
KSYTVTASGSWATQQGKISLNTDALKNS
>lcl|NZ_CP026520.1_prot_WP_084706556.1_2415 [locus_tag=PC41400_RS12345] [protein=ankyrin repeat domain-containing protein] [protein_id=WP_084706556.1] [location=2681397..2682170] [gbkey=CDS]
MFMRINSFIPALIVPVLISAALTGCGMAVSPDPGTAQTPVYQGSQVPETEGIGKMEMPSEKNTQLLEAAAKGDAEKVRTL
IGEGAEVDARDSQGNTAVMAAAINRRHEAVRVLLKAGADVNIRNDRSDNLFLYAGAEGDMELLRLAISAGADPALINRYG
GTALIPASERGHVDIVRELLTRTRVDVNHVNRLGWTALLEAVILGDGGEAHQTIIRLLLEHGADVNLPDGDGVTVYRHAV
SRNFTAIAAMLEAAGGR
>lcl|NZ_CP026520.1_prot_WP_042228273.1_2416 [gene=cysK] [locus_tag=PC41400_RS12350] [protein=cysteine synthase A] [protein_id=WP_042228273.1] [location=2682275..2683210] [gbkey=CDS]
MANIATNLTQLIGNTPLLELVNFGKKQQIEARLLAKLEYFNPAGSVKDRIGYAMIKDAEEKGLLRPGSVIIEPTSGNTGV
GLAFAAAALGYKLIITLPESFSIERRKLLAALGAELVLTPAEEGMSGAIRKAEELGAELPGAFVPQQFENPANPDVHRRT
TAEEIWRDTDGTVDIFVAGVGTGGTVSGVGQALKAKNPAVNIVAVEPADSPVLSGGKRGMHKIQGIGAGFVPGNFDSSVV
DEILTVKNEEALTTARLLARNEGLLVGISSGAAVHAASVLAERPENKGKTIVVLLPDTGERYLSTDLFQEE
>lcl|NZ_CP026520.1_prot_WP_042228272.1_2417 [locus_tag=PC41400_RS12355] [protein=SDR family oxidoreductase] [protein_id=WP_042228272.1] [location=complement(2683307..2684182)] [gbkey=CDS]
MKSKSLEGTIAVVAGATRGVGRGIAAMLGEAGATVYCTGRSVRGNLSNMGRSETIEETAELVTAKGGRGIAVRVDHTVEE
DVKALFERVKLEQNGRLDLLVNDVWGGDPLTEWGKTFWEHSLQDGLLIQKRAVRSHMLTSYYGAPLLVSRKQGMILEITD
GFDYRYRGNLYYSLAKISAIHLAEAMAADLKPHGVTAAAVTPGFLRSEAMLDHFGVKEENWRDAIAQEPHYSESETPFYI
GRGIAALAADPDKFAKSGKVLTSWGLSDEYGFSDVDGRRPHWGRHAAEHGL
>lcl|NZ_CP026520.1_prot_WP_042228271.1_2418 [locus_tag=PC41400_RS12360] [protein=WYL domain-containing protein] [protein_id=WP_042228271.1] [location=2684484..2685452] [gbkey=CDS]
MRADRLLSLLLLLQNQGKMSSRELAEKLEVSERTIFRDMEALSAAGIPVYAERGSGGGWSLTEGYRTRLTGMKAEEIATL
LIANDSALLSDLGMSSEYDSAFQKLLAASPASMRKETERVRQRIHIDGAGWHRSAEQLPHLPLVQEAVWEERKMRIRYRR
DDRIAERIIHPLGLVAKGSIWYAVAEAEGEMRTYRISRLLEAELSEEHFTRPATFDLAGYWEQSLKEFTAKLPRYPAKIK
IREPEFHRFSRERFAKVRNPSEADNGWVETDTEFNTLESAIGILLGYGSSVYVLAPEELRSGLIAVTKGILELYDNKGNK
EL
>lcl|NZ_CP026520.1_prot_WP_241688845.1_2419 [locus_tag=PC41400_RS12365] [protein=hypothetical protein] [protein_id=WP_241688845.1] [location=2685569..2685784] [gbkey=CDS]
MDSGASATAAAVNPFVLIQMLMMAGYGVLLVLGIVLLIVLIRLALKANKALDTYMASKPNTAVLPPETKES
>lcl|NZ_CP026520.1_prot_WP_042228268.1_2420 [locus_tag=PC41400_RS12370] [protein=hypothetical protein] [protein_id=WP_042228268.1] [location=2685785..2686198] [gbkey=CDS]
MVPLKIRLVYNAIFLGTFGVPFYFVYSFIFFDSTHPAQTIYFVLISLAAMNIRLREKLLSVPQKESRKKKIFVGVLGLSM
LTGVFAYYEGWQTALQKGPLGMLLIMGGLYTGHLLLEKLIAARFRADNPPEESSLSS
>lcl|NZ_CP026520.1_prot_WP_042228265.1_2421 [locus_tag=PC41400_RS12375] [protein=SUMF1/EgtB/PvdO family nonheme iron enzyme] [protein_id=WP_042228265.1] [location=2686551..2687231] [gbkey=CDS]
MSDERKAYIDHSMVNIPGGEIELRDDRVHTKWKAEIRPFLLARFPVTEGMYKAVTTNSAYSPDENSKPAVNISWEEAVSF
CNLFSRQAGLKECYSISGDGLTILCDWEADGYRLPTEAEWQYACKAGTAGYRYGDLDEIAWYNENSGGGAHEVGTKEPNA
WGLYDMLGNVWEWCWDVYDEKVYGSYRIFRGGSWAEEARGCGATCRRRSHPTFGIDDLGFRLARSG
>lcl|NZ_CP026520.1_prot_WP_042228264.1_2422 [locus_tag=PC41400_RS12380] [protein=Lsa family ABC-F type ribosomal protection protein] [protein_id=WP_042228264.1] [location=2687641..2689119] [gbkey=CDS]
MSLIRVSNLTFAYDGSYDNIFEEVSFQIDTDWKLGFTGRNGRGKTTFLNLLLGKYEYSGTISAQVQFEYFPYPVDHKENL
TLDVIRDIFPDYVHWELMRELSLLKVSEDVLDRPFDSLSNGEQTKVMLATLFLKENSFLLIDEPTNHLDMEARKLVSSYL
RNKNGFILVSHDRAFLDHCVDHILSINKTNIEIRKGNFSDWWTNKEMQDEFELAENEKLKKDIKRLSDAAKRTNNWSHEV
EKTKNGTRNSGSKVDKGYIGHKSAKMMKRSKNIEQRQQSAIEDRSHLLKNIESSESLKISQLVYHKSQLAELDHVSIQYG
DKTVCSDISFTVEQGERIALSGKNGSGKSSILKLILGEDIDYTGTFRKGSGLKISYVSQDTAHLRGSLSDYARENGIDES
LFKSILRKLDFARVQFEKDISSYSGGQKKKVLIAKSLCENVHLHIWDEPLNFIDVISRMQIEELLLEHKPTLLFVEHDRE
FCDNIATQVVEL
>lcl|NZ_CP026520.1_prot_WP_042228263.1_2423 [locus_tag=PC41400_RS12385] [protein=transcriptional regulator] [protein_id=WP_042228263.1] [location=complement(2689223..2689882)] [gbkey=CDS]
MNRNRTHKDASTSTRRAIIDLLKQRGGTDVVALSSEFSLSGMAIRQHLNALKEEGLVTYEEEARPMGRPTKLWRLTPAAD
RFFPSGYSELSVSLINSMKEAFGNEGLDKLLTVRNKNMQEQYLQHLGGAAGVSEKLDKLAEIRTIEGYMAEVKDQGDGSF
LFIEKHCPICEAAAACTGLCNNELHLFNTLLGEHVLVERVEYILAGGGKCVYRVRENNL
>lcl|NZ_CP026520.1_prot_WP_042228261.1_2424 [locus_tag=PC41400_RS12390] [protein=MFS transporter] [protein_id=WP_042228261.1] [location=2690049..2691407] [gbkey=CDS]
MSSSSQSIFSPRYFALSIGIILSVMAVGFEGLSVTTIAPSIAGDLNGLDLFGWIFSTYLLAQIIGTLVVGRIIDRQGPAT
PFTFALILFVVGLIAAATAGDMYTMIGARAMQGLGAGAMMTCVYTAISLSYPDELRAKILGAFGTAYVLPSMLGPYVAGL
IADQWSWRFVFWGILPILIVSALLSLPAFRKLKTQGTQGDSGASATWLAVLLTIGTGIFLVGLSMLPIFTGFVLVLIGLA
LMGYPLLNLLPKGTLAFRRGMPAILATRGLFFAAYASTQNFLVLALIDVKGITPSQAGLIVASAALSWCLIAYLQGRWDA
ADQGRGRHTRIILGVLLLAIGIALVFWVPVVTVAVAVIGQIIAGIGIGLAHPISGVVAFSQTGNGDVGETSANLQFADSF
TPGVVIGIGGSILVVCQAAGMSLQSGLIVTMGFHLLLIVTSLFASTRIRPAN
>lcl|NZ_CP026520.1_prot_WP_241688846.1_2425 [locus_tag=PC41400_RS12395] [protein=iron-sulfur cluster assembly accessory protein] [protein_id=WP_241688846.1] [location=2691549..2691908] [gbkey=CDS]
MMIGVSEAASEKIVEILSSANAQNSFLRIGVEEGGCSGLSYTLIADEQKTEGDILLNKGEFSLLVHTNSIQYIEGLQIDY
EERGMIGGFTLNNPNAKASCGCGASFRMENYRGEVKKCD
>lcl|NZ_CP026520.1_prot_WP_042228260.1_2426 [locus_tag=PC41400_RS12400] [protein=GNAT family N-acetyltransferase] [protein_id=WP_042228260.1] [location=2692156..2692683] [gbkey=CDS]
MNEELRELNLEEGSDIFEMTREIGLGANGFVNSFYSETRDEFYNLLLKNYNMARAKQLPEGLVPQTVYWLYVGGRPVGYG
KMRHALNGQLKKRGGHIGYVIRPTERGKGYAKLLLQLLLREAAVKGIHEALLTCNEDNAPSRRVIESCGGVLQDTIDGIC
RYHVPAGRVLGSTGT
>lcl|NZ_CP026520.1_prot_WP_042228259.1_2427 [locus_tag=PC41400_RS12405] [protein=family 10 glycosylhydrolase] [protein_id=WP_042228259.1] [location=complement(2692805..2694436)] [gbkey=CDS]
MNKKHIAALVALAFTAALLVFLTGSLGGRKSLLAQTLSAAGSKAGSITQAENQALSAPRQLRAVWIATAWNIDWPSSQGL
SAAKQQQEFIAMLDEAKNVGMNAVIVQVKPNADSLYPSEYGPWSDVLTGKQGQSPGYDPLAFMIEEAHKRNLEFHAWFNP
YRIGKTKDTAKLAADHPALKHPDWVAVYAGTRYYNPGIPEVRDFVVAGISEVVKKYDIDAVHLDDYFYPYPEKGKDFPDD
SAYQKYRKSEFGSKPNWRRDNVNRLVEAIHAAIKKEKPYVKFGISPFGVWRNQNADPTGSETTAGTTNYDTLFADTRAWI
NAGWVDYIAPQLYWNFGFKPAAYDKLADWWIKETEGKPVQLYIGHAPFKIGQHEPPGWQNPDELPDQLKFNARSAGKISG
SILFSLKDIRRNPLGFRDRLNTELYASPALVPPMPWLGSAAPQPPKALTAEGTADGVRLDWRDGGGNPAYYAVYRAEGSG
SEAARLVGTVRSGGSGAQSFTDRAAPAGKPLRYVVTALDRLHNESTTSRTASADVPGRFAKAK
>lcl|NZ_CP026520.1_prot_PC41400_RS12410_2428 [gene=hmpA] [locus_tag=PC41400_RS12410] [protein=NO-inducible flavohemoprotein] [pseudo=true] [location=2694630..2695846] [gbkey=CDS]
MLKPETIAIIKSTVPVLEVHGKAITTRFYEKLFKSHPELLNLFNHGNQKQGRQQTALANAVYAAAVHIDRLHEIMPAVKG
IAHKHVSLGVTPEQYPIVGENLLAAISEVLGDAATPEILGAWEEAYSVIAAAFIGVETEMRMRRSISRGAGRDSASLRCR
RKRRKAGTSRRFTWCRRTGERWHLSSRVNM*PCR*KSPDGNIS*TGSTACRTVRDSLITGSR*RRKRARIPASRESRRLI
CMKPCGKATHFCFPLPPETLRLTGPRIHRSCSCPRESA*RR**AC*ERSGPSSRRGPLRSSTALPTEDCTP*RMR*LRRR
MKPVCARISATRTRRKKTKPLAPFIRKAG*QRRS*RK*RRKPADMPTIIFAAPFRLCAT*SGTCRSWVFGRNPSTMNFSA
RPLPC
>lcl|NZ_CP026520.1_prot_WP_042228257.1_2429 [locus_tag=PC41400_RS12415] [protein=Crp/Fnr family transcriptional regulator] [protein_id=WP_042228257.1] [location=complement(2695889..2696386)] [gbkey=CDS]
MILHKGEVLFRQGEQGPLFRIRSGLFKVVRLQENGEQFLFNILNPGELIPHHSLLTPKEYHGTAIALITSEVEVIPAAQW
YAGLEENPLRYRDAALVLQDKLRAMQQRIDQLTALTPARKIELLGQWFARYVGDIPLSDVLTQEEIGQLVGLRRETVNRL
LRKQR
>lcl|NZ_CP026520.1_prot_WP_009675561.1_2430 [locus_tag=PC41400_RS12420] [protein=hypothetical protein] [protein_id=WP_009675561.1] [location=2696520..2696720] [gbkey=CDS]
MATTSLDALTTGMFVTSLLLILFIGSMVSMSIMRFFQKKKVQGTVYLVVGLAALVTFIVMASGWSS
>lcl|NZ_CP026520.1_prot_WP_129112560.1_2431 [locus_tag=PC41400_RS12425] [protein=NCS2 family permease] [protein_id=WP_129112560.1] [location=complement(2696855..2698153)] [gbkey=CDS]
MVRFLDRRFGLKAHGTTVRRELMAGLVSFVTSVYIIIVNSAILADAGIPVQAGMIATIATSFVGCWLIGWWSNTPLVIVP
GMGINALFTYTVVHTMGLSWQEALAAVFVSGFLFCLVAFTGLAGRISQSVPHTLKEAIGVGIGLFLTFIGLQKSGIVVPS
ASTFIALGDLGSPEALVTLLTLALMAVLYVRGVNGSFLIAIVFGTAVAWLLGLINPAAEGPGFGLRDAAAVFGALSFGGL
WSLPFWIAAFSMAMVIVFENLGLIHSQTASFGRPDTYRRAFQAAAVSNAAAGLLGSSPTVSAVEGSAGIASGGRTGLTAV
FTGVLFLGSYFAVPLLQAIPDSAIAPVLIILGSLMLGSIKAIPFEDFSEAFPAFLVIAMIPLTYSIVDGIAFGFIAYPVV
KLAAGKGRQVSATLYVIALLFLLYEILQVMNK
>lcl|NZ_CP026520.1_prot_WP_042228255.1_2432 [locus_tag=PC41400_RS12430] [protein=LysR family transcriptional regulator] [protein_id=WP_042228255.1] [location=complement(2698432..2699328)] [gbkey=CDS]
MEILQLQYFRTVARLQHMTKAAEELHIAQPALSKTIARLEDDLGVPLFDRQGRKITLNAFGKAFLKKTEVALNALEEGRK
EVEDLAGMEQGSISVASPVLNRLSGPLGEFLALYPHVRFRLTQAPTEELAEMLESGEADFCFTAYPVGRPGIIEEPILSE
EVFLAVPPGHRLADRTSITLSEAAGESFIGYKKEHVFRKRDDALFRQAGMTPDFACEVNEPSAKAALVQAGLGVAFVGAC
NRSEEPPTALMVRIPITDPVCLSSYQLAWHEKRYLSKAAQKFRDFIRHYFSAGTLTEI
>lcl|NZ_CP026520.1_prot_PC41400_RS12435_2433 [locus_tag=PC41400_RS12435] [protein=NADP-dependent oxidoreductase] [pseudo=true] [location=2699499..2700414] [gbkey=CDS]
MKAMGFTKFGSPEVLEPMELATPQAGEGEVRVRVKAAGVQPVDCAIRTGFRPQGVPVSLPHVTGGEFAGIVDQTGSGVTA
FAPGDEVLSFRMQYTYAEYVVVPQEQLVRKPTAMPWEAAGGLSGAGQTALTALEELAVGPGDTLLINGAAGGVGTVAVQA
ARAKGAAVIGTASPANHDYLRSIGAVPVTYGPGLAERVRALAPGGVDAALDAGGSGRAACGGGTRSRPEPDRDDGRLRAC
R*TRRPLDPQQTVGAAAGRAG*AV*PGTAPHSHSESVPFEPSRRSAQRDGDRSRPG*NRADAGRL
>lcl|NZ_CP026520.1_prot_WP_042228253.1_2434 [locus_tag=PC41400_RS12440] [protein=DinB family protein] [protein_id=WP_042228253.1] [location=complement(2700552..2701019)] [gbkey=CDS]
MKEIAARIHDRTYEIASLLEPVDETLLARRPGADKWSVNEILGHLCDSALNNLPRFIVTRYAEQPAVFQPCKQDEWVAAQ
DYRHTPSEEIITLWGSLNRQAARAIASITPEQGDLLCKIGEHEPVTLRWLIEDYEEHLEHHLGQIVRLKERLEGV
>lcl|NZ_CP026520.1_prot_WP_042228252.1_2435 [locus_tag=PC41400_RS12445] [protein=aldehyde dehydrogenase family protein] [protein_id=WP_042228252.1] [location=complement(2701215..2702699)] [gbkey=CDS]
MNELTLTVRPKIQRFLNGTKRLFIDGEFVDSVYGSTFTTPNPATGDILAVVSEATEEDIDLAVKAARRAFDEGPWPGLSA
AERSRLIYRLADLVEDNLEELAQLETLDNGKPIAETRTADLPLAVEHLRYFAGWSTKIVGQTIPVRGGYFNYTRHEPVGV
VGQIIPWNFPLLMAMWKLGAALASGCTVVLKPAEQTPLSALYLAGLIQEAGFPPGVVNIVPGYGETAGQALVGHPQVDKV
AFTGSTEVGRSIMRQAADRLKRVTLELGGKSPNIILPDADLSRAIPGALSGIMFNQGQVCCAGSRLFIQKKMFDNVVADL
VLYSRKLKQGAGLSPDTQIGPLVSEEQQNRVLGYIERGLEEGAELLTGGTNPFEEGYFVAPTVFAGVQDSMTIAKEEIFG
PVVTALPFDDLDDVIARANDTDYGLAAGLWTENVKNAHYIAGKLRAGTVWVNCYNVFDAASPFGGYKQSGLGREMGSYAL
NNYCEVKSVWINMN
>lcl|NZ_CP026520.1_prot_WP_164977123.1_2436 [locus_tag=PC41400_RS29765] [protein=hypothetical protein] [protein_id=WP_164977123.1] [location=2702943..2703098] [gbkey=CDS]
MLLSRDQKELILTVLKKENKRILSGHKGPLLKKTIADLEQALRNEAINDKR
>lcl|NZ_CP026520.1_prot_WP_009675610.1_2437 [locus_tag=PC41400_RS12450] [protein=hypothetical protein] [protein_id=WP_009675610.1] [location=complement(2703207..2703434)] [gbkey=CDS]
MKTIIVFNNQKIPVFYTEASKKALSLLKIALEAKWNNGKRAVKTCLENLISIEIEGTEAILHSRRESDTLALSLY
>lcl|NZ_CP026520.1_prot_WP_042228250.1_2438 [locus_tag=PC41400_RS12455] [protein=NUDIX domain-containing protein] [protein_id=WP_042228250.1] [location=2703736..2704194] [gbkey=CDS]
MRTPIFYGSVHVFLSRNGETLLLKRKNTGFMDGKWSVVAGRIDGGEEVKAAAVREAKEEAGIDIRPEDLEITGIMHRKNT
NSEWIDFFLRARGWQGEIGNMEPHKCEELRWFPEDALPADMVPYVREALLNRQESLWFDSFGWAGVPVQAAP
>lcl|NZ_CP026520.1_prot_WP_053228680.1_2439 [locus_tag=PC41400_RS12460] [protein=hypothetical protein] [protein_id=WP_053228680.1] [location=2704554..2705027] [gbkey=CDS]
MSTSILLQTLKSFVEQSVRDTGAGDRPVRVHVGWLPGAEKFPPAKPDEPAQLLPEGLFPFVLLRALEGEDGNEEGSVKVR
LHFGTSSTDPFGYADVLNLIEKVRQAVLKKEIIGGMFELQRPLKWKLANTQAYPQWTGEMVTEWTVPSPVRENLVYD
>lcl|NZ_CP026520.1_prot_WP_042228248.1_2440 [locus_tag=PC41400_RS12465] [protein=phage tail sheath family protein] [protein_id=WP_042228248.1] [location=2705134..2706582] [gbkey=CDS]
MSERHGIYTSEVPFSVMTPVSPQATIPVVFGTAPVHLSKLAAAPVNTPILCKSWDEAVDAFGYSEDWESYTLSEFMYSHF
RLYKQAPVVLINVLDPAAHKTSAAPSEAVVTAGMAVVSAEGILPSSVTVQSADGATTYSKEKDYTLAFDKKGRLMLSVVK
GGAISPAATALKVGYDKLDRSKVTAADLIGGTDAATGEVTGLDLIKQVFPRFQLVPTLVLAPGFSHDPQVASVLAAKAES
INGHFKAMALTDLPADRPFTDLAVFKKENGYASERQIAAYPNVTKDGRTYRFSTHLAGVMCRTDLESDGVPHRSPSNKLL
QADGTVLDNGTELPLGPDEAEYLNNQGIFTALNFVGGWKTWGNRTAAFPEITDPQNAFIPVRRMFDWIQNSVILTYWKHV
DDPANKRLVESVTDSLNIWMNGLQASGYILGGRVEFIKEENPATDLMNGKLKFHIFVTPPSPAQEIQFKIEYDAQYLGAI
QS
>lcl|NZ_CP026520.1_prot_WP_042228246.1_2441 [locus_tag=PC41400_RS12470] [protein=phage major tail tube protein] [protein_id=WP_042228246.1] [location=2706597..2707118] [gbkey=CDS]
MAKQVRDKIVGYTVYKNGSQFLGVATVELPSLEYLSESVKGAGIAGEVESPTIGQFGPMTCSLTWNTMDPAALELLAPEA
HALDFRASQQSYNTATGIYATEGAKITVRAFPKTGTLGSLEVGAAMEGSTEFEVVYLKVSVGGKVMLEVDKFNYIFVVNG
KDQLASVRADLAL
>lcl|NZ_CP026520.1_prot_WP_053228679.1_2442 [locus_tag=PC41400_RS12475] [protein=phage tail assembly protein] [protein_id=WP_053228679.1] [location=2707157..2707504] [gbkey=CDS]
MSENKTKTPAAEAGQTETVDERVYKLSRPFTFEGTEYTELLLDFDSLTGGDLLSCEGQMNAMSSGNEVVPVKQISQTYQA
LVAARAAKVPMDLIKALPGKDFSKILVRSQNFLLG
>lcl|NZ_CP026520.1_prot_WP_156972804.1_2443 [locus_tag=PC41400_RS29770] [protein=hypothetical protein] [protein_id=WP_156972804.1] [location=2707507..2707659] [gbkey=CDS]
MGREGGRTLRKLVASFSVATKTPIPYWLSLPVAELLEWTEPIQEVTAKRG
>lcl|NZ_CP026520.1_prot_WP_129112477.1_2444 [locus_tag=PC41400_RS12480] [protein=phage tail tape measure protein] [protein_id=WP_129112477.1] [location=2707666..2710074] [gbkey=CDS]
MANGTNLPATVNLQLDTASVTNVTKMIKDVQYTLQQLVIVNIPPDMLKPLSEEVKETDSALKKAKKTLDGFANGLKTIKG
IQASFMGVVTKIDEFNVAMKQVQASTGVSKTEMVELEASAKALYKQNLGGTWNDLAGSLSTVKQLTNQQGAALQETTKNA
IVFKDVFGKDVKSSAASAEYMMKDFGISSTEAYNLFAQGAQKGLEKSGNLLDSVNKYGAHFKTMGFSANQMFNTFGLGTK
LGVSGLDTIGGAMKEFSTTSTSGSKDSVAAFKALNMDAESMTQIFSKGGPEAQAAFNKVATSLASITDPMKRNEIGIKLF
GTSFKGLQQDVLMAMGSARSEFDMTKNTMAQLSTVKYDTLGDAFNSIGRQLFVEILQPIQNMAMPVLSQFADSLIVAIPA
VKTAINEVFNTVMTMGPQIGSFITGFLEEHQKAILGVAGVLTLVFGPALVKTGLQALSAGYQVVTTFVKSIVKAGLESVV
AAGKMTGGIIAAMYKFAAQGWQSALALGRNTIAVLRNVAAMALLKAQQIATTVILWGMAAAQNGVNLAFLANPITLVVIA
LAALVAGGIALYKNWDTVKAKAYDLWTAMQGTFGSIGAFFSNLWTDIKNGAIDGINFVINAINSGISKINGMSINVPSWL
GGGTFGMNIPPIQTIAKSTPSQGATNKGNPGSGTASPPKGLILKVIRSSGQAIPLLKDLHLRLMQKAELSQVPSSPGSAK
EETARSSFRSTIRTAAADCMRPQAGCWAFRLPEVRPEAGPLFLTRRITSTEMPIRPLSSRWSRERDGILKENSTPIYVKK
RG
>lcl|NZ_CP026520.1_prot_WP_042228242.1_2445 [locus_tag=PC41400_RS12485] [protein=tail protein X] [protein_id=WP_042228242.1] [location=2710082..2710291] [gbkey=CDS]
MIYTTIQGDTWDGLSYQFYGSERFMPLLLNFNEKHAEIVVFPAGITLEIPPKPAQSSDTLPPWKRREPV
>lcl|NZ_CP026520.1_prot_WP_042228240.1_2446 [locus_tag=PC41400_RS12490] [protein=contractile injection system protein, VgrG/Pvc8 family] [protein_id=WP_042228240.1] [location=2710288..2711319] [gbkey=CDS]
MKDVLDARRAELVVEYNGKDITRSLADSLVDFTYNDAAPGSLDDIQLSLEDRLQNWQGPWSPSAGDTIKAEIRTFGWNYP
EEEQKLPCGTFEVDSFDFGGPPDTVSIKAVSLPVDSGIRQEKRTKAWENVTLRTLAGDVARAAGLTLFYSADVDPAYERL
EQTDQSDLGFLNEQAMKEGIAVKVSEGQLVLFDEAEYEKKDPALTLVRGEDAVASHKFGWSTAYTAYRACEVTYSDSAKN
TTYQAMYVPPGAPAEGPVLKINEQVDSEAAAMRLAAKSLREKNKEAGKASLSLSGDVRIASGVTLLLQGWNRFDGKYLIE
SASHKVGSGGYTTDIEIRKVLGW
>lcl|NZ_CP026520.1_prot_WP_042228238.1_2447 [locus_tag=PC41400_RS12495] [protein=phage baseplate assembly protein V] [protein_id=WP_042228238.1] [location=2711321..2711737] [gbkey=CDS]
MASENPVIRAGIVSSVQASEGTVRVTFPDRNQYVTAGLPVVSPMGGWARGNSLPAVGENVLCVFAGSGLQAGFCLGTFFT
AAKSPPGLPSQRGVWFEDGSSVYYDRQSGVLHVKPAGSVKIEGDLIVTGTLTRGGEAL
>lcl|NZ_CP026520.1_prot_WP_042228236.1_2448 [locus_tag=PC41400_RS12500] [protein=phage tail protein] [protein_id=WP_042228236.1] [location=2711734..2712123] [gbkey=CDS]
MIGTLGDIVFIATADTIRTFDDFKRSSSSRWAKHEILGKKARTQWIGPGLDTISFTMRFDLRYGINPRKELDKLVETERS
GKAMSLTIGGKGLGVAFWVITSLEQSWDKVDNRGNILLASASVSLEEYV
>lcl|NZ_CP026520.1_prot_WP_053228678.1_2449 [locus_tag=PC41400_RS12505] [protein=GPW/gp25 family protein] [protein_id=WP_053228678.1] [location=2712120..2712452] [gbkey=CDS]
MKIDYTVDMREPGGFYFGPKSEAEEVIQNLRTILSLPAGSAPFARNMGIPFEELDEPMPVAKARMGGSVMTAILEQEPRA
LVEEIRFYEDETSGSAGRLIPVIRFKLREG
>lcl|NZ_CP026520.1_prot_WP_241688847.1_2450 [locus_tag=PC41400_RS12510] [protein=baseplate J/gp47 family protein] [protein_id=WP_241688847.1] [location=2712458..2713576] [gbkey=CDS]
MNEDAYPFIQEDAAETIQEILAVHEAITGRKLFPADPERLFLMSLANMIVHQRVLINQTARQNLLRYARGTLLDHMGEMT
ETKRLQAAPAVTTVEFTLSMPLNSATLIPKGTRIGPQGGGGALYFVTTEVVEIAPGQLKASASAECAVPGTAGNGFEPGI
LNVLVDPVPFVVSVRNTTESSGGSETESDDAYRERIRTAPESFSTAGPEGAYEYWAKSANPAIVDVGVHSPAEGEVVVLP
LLRGGELPTADVLDAVKEAVNDRRVRPLTDHVTVRAPETVPYDIGLTYWVSRSRAADSAAIQAAVAGAVEGYRLWQKSRL
GRDINPSELIGRVMAAGALRVNVASPAYTELSALKVAQDGKVNIVYGGLADD
>lcl|NZ_CP026520.1_prot_WP_042228234.1_2451 [locus_tag=PC41400_RS12515] [protein=phage tail protein I] [protein_id=WP_042228234.1] [location=2713569..2714132] [gbkey=CDS]
MIELKHASLLDILPPNLKRDETVRAAALSLDGQLRQLSEEIGRLTYYDRLDKLTEEEADELAWQFHVDFYEPGLPLEQKR
ELVRNAFRWHKRKGTPSAVEELVAALFGEGKVEEWFEYGGQPFRFRVITNNPEVTQERAREFYRAVESVKRLSARLERVI
LSQTEELPLYIGGVFHMGEKMTIRQVG
>lcl|NZ_CP026520.1_prot_WP_053228676.1_2452 [locus_tag=PC41400_RS12520] [protein=phage tail protein] [protein_id=WP_053228676.1] [location=2714132..2717599] [gbkey=CDS]
MGAFGGFVLTNKGRNLQAKAQTGVTLNYTRMGVGDGQLGGQSIPVLTKLIGEKKSLAISKLKIQTQGRAVIGAVLSNQDV
TTGFYFREIGIYAQDPDEGEILYCYGNAGTNAEYIPPVGGADIIEKSIDAIVIVGNAANVTAEIDKSLVFASQNDLAESE
KRLRDYAESHLENAPSKAFTLENGLQVVNSERMSRFKLESLKGRTLVNRVGRTGSEANVSIFAKANILISKDAATYTTGT
SSYKITATGTGSNEHYCESPRYFIEPNKSYVMIGDARVGAGNVVAGYRLIIRDAAGNVVSDATTTTASKTGVWSVVGMRF
VTPATAATFEWRGQIFKADGSFPFIPNASTPEHANFDSLRLFEIGPADYEFLTTSGWDKASLRFPYVDDMKHTTGAYIIK
IGENLAPTSNEWERGNAAENEVVFKNSYTADIVYNPNRNVYARFYAPVLAGQSYTVSAKTAPSNAAAYYYYTDYDKNRLS
DVLRGSNVAPPKSAFIEVVLKAIDVNLNPIAGTVTYSDVMVNLGTKSRPFKAREDDYLFFSDVKLASNVDGSVTDEVTER
DGKYWKTSRFREVEVNGSLPWLFNIARAGLKRVFASSAVAEYVSRWQLGSSVLKYDGKIMISGAHMDGVDRYTFEDSDLL
YITIANTDSGWGDNYTPSADEIKAYFNGWRMYYWNSTTQQTELFNNQGTKSWEPIVNTEAGAETVVLPKTKWKNHTPYRL
TYQLSRSFETEIVPEGAITLHEGFNQFEAGTGMIVRERVVAAYDIGSGKYNINAAGYAAMLKNKVKSFFKLYRNRKAISD
YYIVKAGIPATGLEANGDWLTNVDDRNPDKPPFYEVTYLTLNPLSAPLSSIGASLSMNLKTIVDMQAQANADIETRVSTL
ERDAMSIKGGNVETLRINGNAATVALVGKDHTYLEFYPQGNDAGRKGYVGYPSAGFSSMHIQNANPFGNIDILPGTGGLL
RVDGNPVWHEGRLRMINGALEYLDGSTWKAVSGGMEQYKPIIKRQTIVLNGNQTLNAIHFTGKCYLSRIAALTLSGSGGV
GRLRVFVDGSLYYDSTLVVNYVNGIFQNNDVSYNSDSSLAVYQPTLRPSSGWIGYNNSTSSYGRVSDFPNSNALQGSAAI
ISQPIFAKSSIVIELTNTSTSSINMDVDFAGGVFQ
>lcl|NZ_CP026520.1_prot_WP_042228233.1_2453 [locus_tag=PC41400_RS12525] [protein=hypothetical protein] [protein_id=WP_042228233.1] [location=2717596..2718120] [gbkey=CDS]
MSQYRIDGSVVYTIEPYGESYYDLLTHLRLSTDKQPESAEDIVTLTVSIYDYVGEPAQENADVQLTINGKKQTLLVTEGV
SSITLSGEPFYEVTADAEGRRGAALLIGSKPEPLKPPTLEELARENEKLRKEIARQNADFQAYIEFSINEQTRLNEEQAR
MNEDFRNFMESVAT
>lcl|NZ_CP026520.1_prot_WP_174446791.1_2454 [locus_tag=PC41400_RS12530] [protein=hypothetical protein] [protein_id=WP_174446791.1] [location=2718144..2718431] [gbkey=CDS]
MVTYDFLRNTYCRSIYVYGTDRFTQRDGFKGIREEYRTPVQLYFAATYGAELLKEALDKGWINQKEYDETRTIMQETVPA
QPTPSQSHPEHQTGR
>lcl|NZ_CP026520.1_prot_WP_042228232.1_2455 [locus_tag=PC41400_RS12535] [protein=hypothetical protein] [protein_id=WP_042228232.1] [location=2718465..2718866] [gbkey=CDS]
MAIAPENFGEHLARLNMESRQFRADIEELKRCEQKNEADIRQLYAGQARTETLVGQVLTRFDGFETRMFNVFQQMTRDNA
ELLQQVTRDGSSERLEGQKERTSAQQAWMSFSKYVIAVTIAAVIAYIFSFFPF
>lcl|NZ_CP026520.1_prot_WP_084706555.1_2456 [locus_tag=PC41400_RS12540] [protein=N-acetylmuramoyl-L-alanine amidase] [protein_id=WP_084706555.1] [location=2718936..2719706] [gbkey=CDS]
MKKPVIVLDPGHGMNDPGAIGSGLMEKELTLKIASRIANKLRGMADVRMTRTDDTAFDTDKSSDLKARCAFANRLQADYF
VSVHINAGGGTGFESYVYSGTAAGSRTEALRSVIHAKVSAAFAKYGLEDRGRKSGDLAVLRQTAMPAVLLEYGFIDRAAD
ASLLKNEAFLDQLAASTAEGLAEAFALGPVSVSPDKSQPQAQPDFSKGVEAAIAKLHEHGVIDSPAYWNNVMTNGQTPDA
EYTAVLIRRMADKLRG
>lcl|NZ_CP026520.1_prot_WP_053228675.1_2457 [locus_tag=PC41400_RS12545] [protein=phage holin, LLH family] [protein_id=WP_053228675.1] [location=2719955..2720353] [gbkey=CDS]
MAQQLTDLLTPVILALVTALAGVLTVLIHAAKTSLIAWLQARTNAAQQEIIMRVASEAMAFAEAQGLILKGKEKMDAAIV
YAANELQTRGMNVTPDKLLAAIQKAWMEYNPAQNKADHAAERKESDTTPAAG
>lcl|NZ_CP026520.1_prot_WP_053228674.1_2458 [locus_tag=PC41400_RS12550] [protein=helix-turn-helix transcriptional regulator] [protein_id=WP_053228674.1] [location=complement(2720737..2721165)] [gbkey=CDS]
MTLVDTIKGLMKEQGLTQYKLAQRAGIPHSTMSTLLNGSIKNPSADMVARIAEVLQVTTDYLLGLSDLPTHEHSPDQTDR
KETRGQSKNSDPHRNEKEFVDKIELSDEELLKQFSLELDGRELSKSETKNIIAFVRTLRQNK
>lcl|NZ_CP026520.1_prot_WP_042228231.1_2459 [locus_tag=PC41400_RS12555] [protein=ArpU family phage packaging/lysis transcriptional regulator] [protein_id=WP_042228231.1] [location=2721406..2721879] [gbkey=CDS]
MKTSIEMPELDRKKTKTAVEGALLKYRMYKTITFEERTATITSSYTEKFHGPTNVTSDQTASIAIYNVEEPERRRRYCER
IEMAVSRLHPKERLLIEERYMKDDYVKDYAVYNHSFDPPISAVTYSSIRWKAFYKLALILSLGVPGPAGQQREGATG
>lcl|NZ_CP026520.1_prot_WP_042228230.1_2460 [locus_tag=PC41400_RS12560] [protein=MFS transporter] [protein_id=WP_042228230.1] [location=complement(2722045..2723331)] [gbkey=CDS]
MSKHPEQLELLHHPDAGKIEPLWTKSFIMLTFCYFLLFLCLQMLLSPFPTYAKDRFHSGDFTLSLLTSLFAISAIITRFV
TAALMRRVHRNTLLFCGVAFAAASTAAYSFAGSMGALLALRVCFGIGFGMASTVMPTLVSQIIPRRRIGEGVGYFGLSSS
LAMSVGPTIGLAMIGSYGFGPLTAMGTAAAALIVPLLLGSRSIPPQPAAAVVSPAGQTAASAGGTAKRSGFNKKLLLPAL
LNALLSVTYGGLLSFIALYGKEVHLEQIGLFFLFNAITVLVVRPISGRLFDSKGHSAILIPGALILIASLLVLSVATTMP
VLIVSALLYGLGFGALQPTTQAWMLRETPPEQHGTTNSLFYNSIDFGVAIGSMLLGLVASGTSYAVMYRYAAGVMALFLL
VYGVYLLTRRRNAGSSAVPAVTEMEISS
>lcl|NZ_CP026520.1_prot_WP_042228229.1_2461 [locus_tag=PC41400_RS12565] [protein=MarR family transcriptional regulator] [protein_id=WP_042228229.1] [location=complement(2723345..2723809)] [gbkey=CDS]
MTTEPIFNKNGLNDSLGFLLGVAYRKISNSLSQRLKEYDITPEQWSVLYRIYEQDGLIQKDIAAKAGKDKPTTTRILDSL
ESKQFIRKQVGQGDRRSFHIYMTDKGRSLIAETVSIEQQTIRDAAAGLSQEQYDQLITFLRGIIDNLTLTSESE
>lcl|NZ_CP026520.1_prot_WP_241688848.1_2462 [locus_tag=PC41400_RS12570] [protein=hypothetical protein] [protein_id=WP_241688848.1] [location=2723983..2724516] [gbkey=CDS]
MTVLLLFGLVLVLIQIILVRRITFRPIPWFTRMAASARLKSPYAYGGFLFIVNMVIFLVISALVVLTVMMNVAFSLFLFA
GAGAFISFMIWIQMSISRESTKKEKRIIGGVGSAFYWVLTVYLLLQIFLLPPSAPEQDPFMQFVGLLMGVVISLTAAVSC
WVTVYLAKGKPVNPVTR
>lcl|NZ_CP026520.1_prot_WP_042228227.1_2463 [locus_tag=PC41400_RS12575] [protein=amino acid permease] [protein_id=WP_042228227.1] [location=2724861..2726300] [gbkey=CDS]
MEKKQTAEEDEKKLKWWQLSLLGVAFTIGTGYFLGSGLAITMGGPAVLISFVLAAIGTYMVFDVLAKMTAADPVKGSFRS
YAKKAYGRWAGFSSGWVYWSSELLIMGSQLTALSLFSRFWFPAVPMWGFATGYGVLGLGIIMLGTKGFERMENLFAVIKI
SAIVMFLVIAALALFGIIGGSRAPRFPVEASTLFPAGASGLWSSLIFAFYAFGGIEIMGLMALRLQKPEEAPKAGKIMLL
LLAVVYVLSIGLAVTLLPWSVFHTKESPFVLALSHYRLPFIPHVFNGVLIIAGFSTMVASLFAVTTMLVTLAEDKDAPKL
FARRWKKNLPLPALGLTACGMTASILLALLMPGGVYEYITTAAGLLLLYNWLFILITAGKLLCLKAPGQAKRITGIALIL
LAVSGTLAHHGSRPGFYVSLGFIGVITLITRMMSRKWKKAVDSSEGQEQVSPDAEEEWVFKPALSMREKTDVFFRKKSR
>lcl|NZ_CP026520.1_prot_WP_042228226.1_2464 [locus_tag=PC41400_RS12580] [protein=anhydro-N-acetylmuramic acid kinase] [protein_id=WP_042228226.1] [location=2726520..2727704] [gbkey=CDS]
MILGIKEEEVIAIGLMSGTSLDGIDAAVVRINGSGLDTRLELLHFYSRPYDDALREKLKDLCSVARSDVALVCGMNVFLA
ERMAEAALEAVREAGLTMDRIGFVASHGQTVWHIPEGDAADPYLVRSTLQLGDLSVLAKRTGVPVVGDFRPADMAVGGQG
APLVPYGDLILFRHETKGRLLQNIGGIGNCTVLPPQAGPSDVFAFDTGPGNMVIDQVVYQLSGGRHSYDESGSWAAQGRA
DEGLVADAMTHPYFSLKPPKSTGREVFGKAYAADFIARAKALGLADADIVATATAFTARTIAEGCRRFVFSRCRVDEVIV
SGGGAHNRTLLSMLAELLPDQHVTTSGALGVHDDAKEAVIFALLGNDFLRRIPNNLPSATGAARPTVMGKLALP
>lcl|NZ_CP026520.1_prot_WP_164977124.1_2465 [locus_tag=PC41400_RS29775] [protein=hypothetical protein] [protein_id=WP_164977124.1] [location=complement(2728008..2728166)] [gbkey=CDS]
MKFLVWLLKAIFVLPAVLVMELVFGHQGTLFLKETAKKDDTGGEMYRKIEKR
>lcl|NZ_CP026520.1_prot_WP_042228225.1_2466 [locus_tag=PC41400_RS12585] [protein=nucleoside recognition domain-containing protein] [protein_id=WP_042228225.1] [location=complement(2728309..2729715)] [gbkey=CDS]
MATPHATNRPDPIDSLLSMAQTITSGSSIRDEIVSDIYKTSCDICNAAVRYEDKKKLAGTHNLDKILTSKLWGYPIMFAV
LGAIFWITIAGANVPSGWIADFFGVIEGYLTAGFQAVHAPAWLHGILVLGFFRGTAWVISVMLPPMAIFFPIFALLENFG
YLPRVAFNMDRLFKKSGGHGKQALTMSMGFGCNAAAILSTRIIESPRERMLAILTNNFVPCNGRWPTLILLSSLFMAAGA
SGGWHTLGTALFVMGLVLFGIVITLSVSWVLSKTALRGVPTHYTLELPPYRKPQMWKTILLSSRDKSWAVLKRAVIVAAP
AGIITWILGNIMINGTSILAHTAAFFDPFAHLIGLDGYIIMAFLLGLPANEIVLPILLMGYMSSGAMVDADGMDNIKDLF
LQHGWTWLTALNMMLFSLLHYPCGTTLFNIYKETKSTKWTILSALIPLAIAILVTFATAQAARAFGWV
>lcl|NZ_CP026520.1_prot_WP_042228223.1_2467 [locus_tag=PC41400_RS12590] [protein=FeoB small GTPase domain-containing protein] [protein_id=WP_042228223.1] [location=complement(2729693..2730424)] [gbkey=CDS]
MKTYTVALAGNPNTGKSTLFNRLTGLRQHTGNWAGKTVLTAEGEFERNGILYKLIDLPGTYSLYSNSADEEAARDYIIFE
QPDVTLVVLDATSLERNLNLALQVLEMTSRAVVCINLMDEAEKLGIRIDLDRIRERLGVPVAAISARNNKGIGPMLDLVE
KVAEGVLHPEPLPIPYSEEIERQVRELEPFVRKTLGARYPARWIALRLLDGDENLIASLKQRLNGAAPAQGQGVMSYGHT
ACN
>lcl|NZ_CP026520.1_prot_WP_009675529.1_2468 [locus_tag=PC41400_RS12595] [protein=FeoA family protein] [protein_id=WP_009675529.1] [location=complement(2730421..2730675)] [gbkey=CDS]
MSLSELSLDQVKVGSVVRLSRIEVQGGMRRRLLDLGFVPGNRVEVIQRSPLGDPTAFRVSNTTIALRKEESSLIYTEPEG
GDSL
>lcl|NZ_CP026520.1_prot_WP_042228222.1_2469 [locus_tag=PC41400_RS12600] [protein=metallophosphoesterase] [protein_id=WP_042228222.1] [location=2731144..2731656] [gbkey=CDS]
MPSDFFISDHHFGHKLIIDFESRPFADADEMTDVMIQKWNAAVGSEDKVFHLGDFSFLNQERTQEIVRKLNGYKFLVLGN
HDRGRSRSWWLDAGFDEVSEYPVLYKDFFLLSHEPVYMNKHMPYVNVHGHIHGQKYEGKNYFNVCVEHWDYIPLTFEQIR
DAVVVSEEQA
>lcl|NZ_CP026520.1_prot_WP_042228221.1_2470 [gene=coaD] [locus_tag=PC41400_RS12605] [protein=pantetheine-phosphate adenylyltransferase] [protein_id=WP_042228221.1] [location=2731653..2732159] [gbkey=CDS]
MIAVYAGSFDPITLGHTSVIDRATNLFDELHILVANNRAKKHHFTIGQRTELVRLSVPPGKRIKIVAFEGIVADYVNETG
AKAIVRGIRNATDFEYEMQLEQYVRNTTKADTVYLSPYTPNMQTSSSLVRMFIATRKVEPAGAFMVEAAFEQMKMYESEN
NQRRSGIE
>lcl|NZ_CP026520.1_prot_WP_042228220.1_2471 [locus_tag=PC41400_RS12610] [protein=hypothetical protein] [protein_id=WP_042228220.1] [location=2732280..2732693] [gbkey=CDS]
MLIGACVLDAHGEPGENTGGVNAYPAKPRPGSEVYFRLTGYPPQITNASTKSLQSSPSGQILRAFFYSLAFYLSGRDHFP
PYFYYFSFLAINLVFYNNCAYDEIVLGVNHVDDTVSPSFVTVQFQEFIASREKYGHK
>lcl|NZ_CP026520.1_prot_WP_042228218.1_2472 [locus_tag=PC41400_RS12615] [protein=MurR/RpiR family transcriptional regulator] [protein_id=WP_042228218.1] [location=2732680..2733543] [gbkey=CDS]
MAINDNVLIKIRQMKTSLTPVEKAVGEYILENVEEIPHLSIKNLARLTKTSDASVLRFCKTMGYPGYRSFIVSISASLGS
MHEEDANPYRDIQPGDDLSTIILNISRTNSQSIEDTLQVINKSEIARAVDVLRSCRRIAFFGIGASGLVGTDAEQKFTRI
NKMCHTYTDGHSMLTAATLLEKGDVAIFISNSGETVEILDALEIAKKNGACLIAITKYTKSRLADQANILLSISTPEITI
RSGAMGSRIAMLTVVDILFAGVASAEYQHVKKYLAKTHNIVAGKQRR
>lcl|NZ_CP026520.1_prot_WP_042228216.1_2473 [gene=murQ] [locus_tag=PC41400_RS12620] [protein=N-acetylmuramic acid 6-phosphate etherase] [protein_id=WP_042228216.1] [location=2733936..2734850] [gbkey=CDS]
MDEYLARLTTEEINEKTLKMDEYTTEQMLRLMNEQDATVPAAIAAEIPQIVKAVDILHGVLKNGGRMFYIGAGSSGRLGV
LDASECPPTFGIDPLRVQGHIAGGDGALRIAVEGFEDNKDEGIALIERCGVTDKDAVIGITASGSAQFVIAGLNKAREIG
AATIGLVNNKHSKLESVCDVCIAPVVGPEVIMGSTRLKAGTAQKLVLNMLTTCTMVKLGKTYNNLMVDLKASNIKLKDRS
IRIIKAATGVDDKTAAAVLESASMSCKLAIMMIKSGLDAAEAQRVLDQCEGSLKAAIRTTTNVV
>lcl|NZ_CP026520.1_prot_WP_042228213.1_2474 [locus_tag=PC41400_RS12625] [protein=extracellular solute-binding protein] [protein_id=WP_042228213.1] [location=2734883..2736304] [gbkey=CDS]
MKRKLRAAASVLALTMAMSTLAACGDSGKKEGGAAGDPNAKDTITALLPPVSASYQNNFAQMAKDFNALYPNLTLKIEPA
SWEDITQKLDTQVNAGSPPDIAFMGSDGISKYMQQEMMMDIKDVATKEMVEDFDQAPLEYMKKGDGLYGFPGYMEVHAIG
GNREYLEKAGIDWKKVQKDGWTFDEFREAIKKGVVSEGGKTSRYGFVFATAGVASKDYLNILIKNAGMPAPFTKDLKYAY
TSKNYLEVLKDIRTMIDDGSMPKELSSIDAGKRWNMFLTGQTMITGKGLATFENSAKKNNEKLKANDGSAVKGSVPVEYT
VLPVPTFAGQKPQSSVAVDGYVTFRGSKEPTAEHKANVVKAAYFLASGKVAATTNNDLFVAHITKSGKKEAESMTVNRDA
DNKAAVETMLKNAAPARPDIPADLGAKAIKLEEEVIIPKLQALISGEMTPEAMFEAVKSAAVQTFGADGVVKE
>lcl|NZ_CP026520.1_prot_WP_016424014.1_2475 [locus_tag=PC41400_RS12630] [protein=sugar ABC transporter permease] [protein_id=WP_016424014.1] [location=2736391..2737293] [gbkey=CDS]
MTQPAKPVKRRKIRGDGAWGYAFIAVALLVYALFTAYPVISAFIISFQKYRPLGSTFIGFDNYTATFKDPLFWKSVVNTI
VYTVLTVPVALFLSFAVAILILPLKKKLQTTFKAIYYLPAVASGVALSVVWLWIFDPMPSGILNKLISYIGISNQNWLGS
SATAMFSLVLMAWLSSHGTSIIIYLAALLGIDESYYEAAELDGASFIKKLWYIVIPCLKPTTLFLLVTGVIGSFQVFQNA
YLMTGGGPDNATTMVGLLVFNNAFKYFDFGKAAAQSLILAFIIAGISLVQFKFLGKEVEY
>lcl|NZ_CP026520.1_prot_WP_042228212.1_2476 [locus_tag=PC41400_RS12635] [protein=carbohydrate ABC transporter permease] [protein_id=WP_042228212.1] [location=2737312..2738172] [gbkey=CDS]
MALYNNHAGNRAARYSRNTVIIVILTLFALATIFPIYFMLISSFGDPIEAGAVSYSLLPTKFTLESYKFFFDYSEYSIRW
LLNSLIVATTVMVSNVIFASMAGYAFSKIKFKGRAMLFTILLIAMMIPYQVTQVPLYILIVNIFHIQNTYSALILPGLVT
VYNIFLAKQFMGSIPTEIIESAKIEGCNQFQIYFRVILPLSKTVLAVMAILTFMDSWNTFFWPFLVTNTMDMQTIQVGLK
NFRFANTTYFAPMMAGATISALPMFVLFFSLQRYFLEGVTVGAVKG
>lcl|NZ_CP026520.1_prot_WP_042228211.1_2477 [locus_tag=PC41400_RS12640] [protein=GNAT family N-acetyltransferase] [protein_id=WP_042228211.1] [location=2738458..2739426] [gbkey=CDS]
MIETWNPSYSDAVIRLWNTEAVKDGYKELTGRSFNDIFLSSPYFDKDNTFVMREEDGTITGFACGCTGSDLPLGAVAGYI
TCIVLSEDRQTDANYRLLLEALEGRFRQLGKKQADVLFFNPMMLPWYIPDTPGHEHNNAPGVPADSRLYGFLLGEGYIER
AMECAMYLNLSAFSIPDEIRAKEERTAAEGYAVELFDPARHEGIEAMLQGFDNPLWQQEIAAGTANGVPVVIAAHRGQAV
GFAGPVIRQESGRGYFAGIGVHPDHEGHGLGSLLFFKLCEAFKGIGTEYMSLYTGSANPAIRIYEKAGFKTVKQFAVMRK
AL
>lcl|NZ_CP026520.1_prot_WP_042228210.1_2478 [locus_tag=PC41400_RS12645] [protein=hypothetical protein] [protein_id=WP_042228210.1] [location=2739788..2740372] [gbkey=CDS]
MGMEAELERLKRLVFQLLLTTENRTTDFLELITGETMIPSVLKQATATQAPLVAVKEGPRDAVLRESLLRAEKSGLVVSQ
NIAVMYPDFIPLPINLKLAQKKEGIGKIMRDLNIRSTRTVLDFGWKKPEHTTDLNGNPYSVTFADNTAVPYKQYTITFEE
FAMPGLHLLEYFNPRMLEGKEVGAGQGIYNAGVN
>lcl|NZ_CP026520.1_prot_WP_053228673.1_2479 [locus_tag=PC41400_RS12650] [protein=hypothetical protein] [protein_id=WP_053228673.1] [location=2740617..2741111] [gbkey=CDS]
MHSVQEEKDHRRFLKQTIEHFKLGNYFPEELILLAKSDPVSFEWMRHTEHGTIVFLMSTGCSACNMDYAVSFMNSQPGFN
YCILFEGSRESVEELRSELNPQVRIHRCETAKLQAQLNVNAVPYAIVLNKVGQAVGADIFDKIANLNRAAAPLIRVYHNL
NPDK
>lcl|NZ_CP026520.1_prot_WP_241688849.1_2480 [locus_tag=PC41400_RS12655] [protein=ATP-binding cassette domain-containing protein] [protein_id=WP_241688849.1] [location=2741159..2742877] [gbkey=CDS]
MYAFNKIIVLFSLFAYVFSAVQANVSVYLSKLIVDQLTIKNFYLLMGGIVLLILFQLLRAALDAYAQLKSRKMSLMFAIQ
CEGKLIDLVEKTDLIEKEHPRFKGDFSYWSYSNNKYYDTYLIAIGLIKQIILASLSLFFLLHSYWLTGILALLVGMWRGV
YDLLAVRRRVEIDEQLMRNTRSHHYYYDLLTNSETQKEMMLFRLFPYFRKRWLEKKTEYAELQIKLDKLNHQRTFRGELL
SILNTAAVTVITAYLIYKGSLSIGDYVAITMALSMTENNISSMFTSYSRLKEYCTHIAKLKLIEEKNEVAVSAEKNVRSK
PFVFEKDIRIQDLSFRYPNQDQPALNGISAEIKKGEVVAILGENGSGKSTLIKLLLGLYQTEEDSIFYDDVSLKNMDRFQ
MWQKSSAVFQDFIRYMTDVRENVAAGKIGEIEDSEKLEQMIARVGLADTFKHGLNSRLGFLDDGAVNLSGGQWQRLALSR
VFVRDDDDLVVFDEPTAALDPLSEVRLMNDLLGHFKGKTVLLISHRVGVARNADRIIVMERGRIAETGTHNELLERQGLY
NEIWNQQKQWYV
>lcl|NZ_CP026520.1_prot_WP_042228207.1_2481 [locus_tag=PC41400_RS12660] [protein=hypothetical protein] [protein_id=WP_042228207.1] [location=2742903..2743358] [gbkey=CDS]
MTSSANPSRLSDTGLPLGASLPELPDNLKEGMGAAWNQSNTIRVLLFASVYCTHCIDLLPHLQTIAENHPTFSFGLFTTG
DEEDHRSMKEYFGWDFPVNTMDQSDMEAYFEVTALPFMMLTDKSGIIAAKGVIYNADDFEQILRISGIHLT
>lcl|NZ_CP026520.1_prot_WP_241688850.1_2482 [locus_tag=PC41400_RS12665] [protein=hypothetical protein] [protein_id=WP_241688850.1] [location=2743660..2744055] [gbkey=CDS]
MNAQLLVCDQITNNTLLGGHALGTVLNQVSIPALPFVIEMHVLIKLFDLPRDEFVDTALIITNSSGEIIDRTPASVLRNY
RAEDQVPGVDQDVSIHLIVTEPGIIYIRCMVNGEEAAWYPLTIRLQEQKVS
>lcl|NZ_CP026520.1_prot_WP_241688851.1_2483 [locus_tag=PC41400_RS12670] [protein=hypothetical protein] [protein_id=WP_241688851.1] [location=2744102..2744635] [gbkey=CDS]
MIQWSGYVLDMTMSVLFFMPFYIKAIRFRNFRIELYSYGILPRSLVSIAALVVLSVEFLLFLSFAIGLFGGWRELAAIVL
LAAFAFFSWRKTKLTGVDACACYGEIAFLNRSPVYRNIILIGLLVVDLLLPTPEQDGYGQINSLLFVMALSISIGLLQSA
WTKRKRRRHHEHPAVPM
>lcl|NZ_CP026520.1_prot_WP_042228203.1_2484 [locus_tag=PC41400_RS12675] [protein=hypothetical protein] [protein_id=WP_042228203.1] [location=2744610..2745023] [gbkey=CDS]
MSTLLFLCNLVWFLACVLLLFIQKRKERDEVTALIGEIKRLSSRQQSVTRIIFADHKDPAFQKIDSLLSTSADGPETIVV
INAPSWLIAAREKKWTKHEVIDAKRISSASKSGIIVIRGGKYDIYDEAAAYLAYTSS
>lcl|NZ_CP026520.1_prot_WP_241688852.1_2485 [locus_tag=PC41400_RS12680] [protein=AraC family transcriptional regulator] [protein_id=WP_241688852.1] [location=2745442..2747262] [gbkey=CDS]
MDGELFEISGFHVLHGGKGSRLDIYAVEEPFSYYLLYYKAYFPVPLRKEAAQLANNCNPFQLQYGLTPHHPLSLMSKIRL
MHQEWVTQGNLERLHVKALFYQFVYEVLAQVHARNIPAAKADVVDQARRYMQENYTEPVTLEKLAELLGSSPRHLTRLFK
RKTGTSPIDYIIRLRLNKAKEMLLSTDLTLQEIAESIGYPDSYSLAKMFKKQVGIAPVRYRTEQRNRRSCPDLPSDTAVF
DIAGRDRLNYIEYDNHSQYKKELNFPMLYSRRPSAAITLLLCLMILLSACSTGKTGGSANGNSLPSSSPASVAANTDSSQ
TAPSKPSTKMISTAKGDVEIPVNPKRVVVLYLLGDVLAFDVMPVGISDVYKGAAFEKELSGVRSLGEWFKPSPEAVLALE
PDLIIVPSEETYGMLNKIAPTVYIPYEQMTMEERLNKLAEALGKEKESAKLLDNFHKKVEQSKEKLKQAGIYDKTVTIME
GGKGSMAVISSRNYGRGSQVIYEYLGMKAPAVLQRDIDAGSKAASSKNVSLEALNQYSGDYVFRSAYEGMEDLSQNPAWK
QIPAVKEGRLIEISFGLSYYNDIYSLDKQLDFVMDNLLAKAEKRIN
>lcl|NZ_CP026520.1_prot_WP_042228199.1_2486 [locus_tag=PC41400_RS12685] [protein=deoxyribonuclease IV] [protein_id=WP_042228199.1] [location=complement(2747477..2748352)] [gbkey=CDS]
MSLLIGSHVSMSGKTMLLRASEEAASYGATTFLIYTGAPQNTRRKPIEAFNLDAGKAHMKAHGISNIVVHAPFLINLGNT
VNPRTFEYSIEFLRSEIQRTEVLGSNQIVLHPGAHMGAGAEAGIRKIIEGLNEVLTEKRDVQVSLETMAGKGTECGRTFD
ELARIIDGVTHNERLSVCLDTCHIHDAGYNIAENFDGVLHEFDQIVGLDRIRVIHVNDSKNERGSRKDRHENIGHGFIGL
KALDYIVHHPRFDRVPKLLETPSIGPVKYKNAPYKHEIALLRREVAEPIDK
>lcl|NZ_CP026520.1_prot_WP_042228198.1_2487 [locus_tag=PC41400_RS12690] [protein=SMI1/KNR4 family protein] [protein_id=WP_042228198.1] [location=complement(2748621..2749076)] [gbkey=CDS]
MKNHAWKKIEQYYGIVFPSDYLNFLQKVEQHGYNFVENGDVTDWEIRFSELDERFIETNRSLVDEVNPDPKRLIPFAWSV
SSGNNYLFDYRTNLSCPAVVLMDHEEAMVREDAESESETLEEAQELMEQNVRKAADNFAEFAARLQPYTAE
>lcl|NZ_CP026520.1_prot_WP_042228197.1_2488 [locus_tag=PC41400_RS12695] [protein=GNAT family N-acetyltransferase] [protein_id=WP_042228197.1] [location=2749158..2749589] [gbkey=CDS]
MQIVSAADTDFDYLRDRDPHMKESLILPKIRSKEIYMLRNEENVNIGWMRYGYFWDNTPFMNMLWIDEPYRGGGFGKEVV
LFWEKEMTQRGFKLVMTSTLANEEAQHFYRKLGYRDAGCLLLENEPLEIIMTKRLLPTLSSSK
>lcl|NZ_CP026520.1_prot_WP_042228195.1_2489 [locus_tag=PC41400_RS12700] [protein=hypothetical protein] [protein_id=WP_042228195.1] [location=2749641..2750435] [gbkey=CDS]
MDKKAKKILATTFWSAAGWKNRPEVYAGEDFEYAKSKGLMFDPLNTGHDELVKRLRDLHQTITKERVAAAFLHSLSTKKV
HLRSALSSWALTAEMPLHTYGERSAVRPNYSSCGDCNYRGIMSDREYVNADLNVLNFERIKWGGIRLNQLLYCWLDLELF
SKEEAFEVTAEDRTILCSMIEAIQECEDHDSARKLEKRWKDILPSSKNERDVVMEIWGYAGLLVPRDTPRKRRGGDHDFN
SVAVWQGDDRYSREAMEFYFGAYL
>lcl|NZ_CP026520.1_prot_WP_042228856.1_2490 [locus_tag=PC41400_RS12705] [protein=DUF6530 family protein] [protein_id=WP_042228856.1] [location=2751052..2751531] [gbkey=CDS]
MKIPTTLKHKPVIVSENYEQVDGRLARNTDAKGLSLGLAQWNDRGKVDISAKVWRHTGEKWSRQSEELPMHRVLDLAILI
CRSSLYFQDAYRFPKLYDPENPMIDRIGLQGDAMSVSVCEDNPMLDQDIKLFAQALSEDGEMIGERLRVLSRLLEEMGY
>lcl|NZ_CP026520.1_prot_WP_042228193.1_2491 [locus_tag=PC41400_RS12710] [protein=AAA family ATPase] [protein_id=WP_042228193.1] [location=2751554..2752657] [gbkey=CDS]
MNVNQDYYVSPDRQLTDGEQALVWKKPASHKVSEEEIRICQEIKRNWNHGEMKIANILLEGDAGSGKTQLAKALSADLGL
PYTKVTCFADMDKSDIIGAILPVISGDRVEKMEPAEQTVLRALYDSDGFQSAAELVMNVLGGTREEAALTMKQLLNKIEQ
NSGDDVVEYRFYPSEIVRAYRNGYLLEIQEPTVIRDAAVLMALNSALEPDGSINLPTEIIRRHPDFVAVITTNRSYAGTR
PLNEALRDRVQHTEKMDLPTKEIMVERAMSKTGYKNEKVLAILADVIVVLDKTARAHAIKGVAGMRSFFYWADAVAGGAI
AKESLYHKVIYKITTDSEEISLLEEALKNRGLLDVLDQVDAEAKKKE
>lcl|NZ_CP026520.1_prot_WP_241688853.1_2492 [locus_tag=PC41400_RS12715] [protein=nitric oxide reductase activation protein NorD] [protein_id=WP_241688853.1] [location=2752936..2753763] [gbkey=CDS]
MSGSIHNKVKLIVHRPDYDNDSEREYARLSEELMPVVREIARKTLPLLEHETSSEFARRQYYGSKFQADSIAYRDFRTFA
KKLPPTDSPSLVVGLRVDESASMSAFGRLEAAKRAVIAVYEFCQICRIPVLIYGDTADASRLEQMSIYAYTDLDKTDSND
RYRLMGIRARSNNRDGMALRIMADRLAASPQQTKLLISISDGQPKAMDDYTGTYAVRDMQQTMAEYERKGITFLAAAIGQ
DKDVIHQIYGSERFLDITSLPDLPAKLVRIIARYL
>lcl|NZ_CP026520.1_prot_WP_042228191.1_2493 [locus_tag=PC41400_RS12720] [protein=GIY-YIG nuclease family protein] [protein_id=WP_042228191.1] [location=2753802..2754161] [gbkey=CDS]
MDKNKRKELLEEYKQVKTYMGAIKILNKTNGKLYVDTYPNLKNKWITLQAQLDMGRFANSQLQKDWKELGAENFAFEVLE
QKEASEITDMRWERKQLKKRWLEKLQPYGDRGYNKPPEE
>lcl|NZ_CP026520.1_prot_WP_241688854.1_2494 [locus_tag=PC41400_RS30280] [protein=discoidin domain-containing protein] [protein_id=WP_241688854.1] [location=2754285..2754716] [gbkey=CDS]
MRKFKALVLAAAVGIAFSSSVHLPSAAAAQPTSIAIPTNLALDGKVSASGEFAPHQGKDKAFDQDVFSKWLTFSSPAWLQ
YEFSAAKVVTSYTITSAEDEPYRDPKSWVLKGSNDGSNWTQLDTRQNQSFTSRHQAKTYSFTN
>lcl|NZ_CP026520.1_prot_WP_241688855.1_2495 [locus_tag=PC41400_RS12725] [protein=basic secretory protein-like protein] [protein_id=WP_241688855.1] [location=2754774..2755883] [gbkey=CDS]
MQLSEIKLFDGSAKTWSTTKPTVTASGEHTPTNIKENLVDGTSVTKWLIKDDTAWLKFDFGQQVTLDGYALTAANDRPER
DPKSWVLQGSNDDKTWTSIDTKSDENFNVRHQRKHYLLTNNTKAFQYYRLNNIKNHSGNSIQLGEVEFSRTNDIGHAVNP
VIEVKNLDSSGNGSLFTKALPNAEKEILTVARKVHELLYSNPADAPVDVKKILVTIEDVPGVAWTSGDSTQKTVGFSSQY
LKKFVASPSKSLREEIIGILYHEIGHVYQYSNFDVEAIADSLRFETGYHDRYSATKGGTWQTKGTANFLRWVEDSKKRGF
IRELNAAPIPYNIIGNANEVQLWKENQFKVITGTDVNTLWSQYQATLSD
>lcl|NZ_CP026520.1_prot_WP_053228672.1_2496 [locus_tag=PC41400_RS12730] [protein=AraC family transcriptional regulator] [protein_id=WP_053228672.1] [location=2756274..2757347] [gbkey=CDS]
MIISIHLKMCGPPKRAASDRNGVEFMQSAERYCEIANAENSNCALTLNLNEMYRTCSFEQVVKVPEEMGKGYWKCIRTYS
SIDLVICDVRFTKNMTLSSREGGNNINLGFCLGDSIRWSVEGRKGEFGLDPWEVSAFGSIQASNRCQYDVDSHFQGLSLK
LNHMESIGPLHHLPLHKISSALSGNSRLFYNSQMTSGMKRIVHDIIHCYYHGDIKHIYLEGKILELIALYLYEVILEKNT
SSPPLGLSRTEVASLHLAKDILDGDLMSPPGLEALAKRVCLNEFKLKKGFKLLFGMPVHAYVIDRRLEAAFHMLEEGRMK
ITEAAYAAGFGKTGHFSEQFKRKYGVNPSEYFRQFRR
>lcl|NZ_CP026520.1_prot_WP_042228189.1_2497 [locus_tag=PC41400_RS12735] [protein=ABC transporter substrate-binding protein] [protein_id=WP_042228189.1] [location=2757518..2758465] [gbkey=CDS]
MKRVIVLAILCCMLVLTACGNQETRDGENQQKFKEVENAKGGKVKIPIHPKRIADLSGSTEELLLLGIQPIASGNTDFGN
RAVLSPTIKNLLSADTVNLGWYAEPISLELVVSAKPDLIILGTLFNENLYEQLSKVAPTVIVPYPYYEWRNRFTFLADLF
GEEEKKDKWLSEYDRKVEEWSKKLEPVLGNETFAVIETYPKNLVIYSSSGTAELIYTDLALKRTDGIPEPESWGGREINL
ESLATINPDNLILMENSENTMNDSKVWNNLKAVQKGNVYKITNIDNYNYSFTAIGRMELLDRLGKQILKNHKSNE
>lcl|NZ_CP026520.1_prot_WP_042228188.1_2498 [locus_tag=PC41400_RS12740] [protein=ABC transporter ATP-binding protein] [protein_id=WP_042228188.1] [location=2758568..2760406] [gbkey=CDS]
MSFKSEVSGLLHIAGEKKGLLTVASVFSFFSSLLQLVPFVAVYKIVEELLNHAQSPEGADKELILYWGGVAFIACIGALV
ALYIGGMCSHIAAFNILYQLRVRLAEHVAKVPMGYHTNTTTGELKKIIEVSVEKIEKFIAHQLPDIVSAVMIPILLMGYL
FWLDWRLALVLLIPIGFGFWLQTRMFTSVRGQTAYRGFQYAVEEMNATGVEYVRAMPAVKIFGVTAESFLTFKQSVAKYR
DISLKITDLCKTPYSLFFVVVSSLFTFIVPVGILLGNGHSGSQSFAITFILFLIIAPSLSVPLLKLMHAGSGMREIVEGN
KRIEAIFSEAMVTEPVSPQVPTAHDIAFKQVSFAYETKESKDFKPVLNGIHFVARAGEMTALVGPSGGGKSTIANLLLRF
WDVQEGDITIGGVPIREMGTEKLMDTVSFVFQDVHLFYDTIEANIRMGNTTASNEEVIAAAKTACCHEFIEKLAYGYDTK
IGEGGTYLSGGEAQRIAIARALLKNAPILVLDEATAYADAGNEKKIQRGLAELVKGKTVLIIAHRLSTIRAADQILVVKR
GEIAERGTHDELRKLNGLYEQMWQAHISAASWRLGAKDNSPAHAVAAKEGSG
>lcl|NZ_CP026520.1_prot_WP_042228186.1_2499 [locus_tag=PC41400_RS12745] [protein=ABC transporter ATP-binding protein] [protein_id=WP_042228186.1] [location=2760406..2762145] [gbkey=CDS]
MNRYFTNISGGNIRNLWPSSFAALLDGISKVIPAALLVDIFHTIYSSFAEPEAGLSIGRLWVVCAFLFFWLMVSYGAYAL
LYDKSYKAAYSLAASGRLKLAEHLRHLSLGYFGKRDPGDVTNLLLSDYSQVEHTISHNVPQLISAVILPLLALSGLVFLD
WQMALAMFVAVPFGVLLLWLTDALQARLSETHVKAKNEAASRLQEYLSGIREIKAHNMGGKRFERLRLSFDRLMRASIRL
EGVMGPLIMGAMLLTRSGMTIMIFAGSYLLVGGTLSLPVFLLFLLVGTRVFEPLTVVLMNYGEMRYSAYSAQRIMDVQQE
QPMQGTESIQPQGQIVFDRVIFGYDDNALIIRDLSFTIEPQTITALVGPSGSGKSTVTRLIARFWDIQQGTIEIGGQSIR
HANPEDLLKHISMVFQDVYLFQDTIGNNIRIGKMDATQEEVEAAAKRACCHEFISQLPQGYDTPVGEGGSTLSGGEKQRI
SIARALLKDASIVLLDEATASLDPENESAVQKAINELVADKTVILIAHRLKTVQHADKILVLDQGQLVEEGTHMELLKRK
GLYADLWSLQQESDGWRVK
>lcl|NZ_CP026520.1_prot_WP_042228185.1_2500 [locus_tag=PC41400_RS12750] [protein=MATE family efflux transporter] [protein_id=WP_042228185.1] [location=2762182..2763570] [gbkey=CDS]
MSQTTVDEKGKQQYFILNVNLWKVMLQLSWPAIIAMVLYGLNAVIAAVFVGRYIGETALAGVSVAYPLTQISVGLGSLIG
VGAGSVLSMAIGRQDKRTQERLLGHVNYLSLIVTVIYMILGLLFSTQLIRMMGGEGEVLFLGDRFFRITVIGSFFWIYGL
AANMIVRGEGKMKSAAVMMGIGLAADILFNYILVVILDSGVEGAAWATNISMFVYTLLGWIYFGKGFASFKTKVFSIYRD
AATVKSIIGLGLSSLIMNVMSLVQGVVVFNALAKFGTMLDIAFYGVVYRVFTFLLTPIFGLMRALQPVIGINYGAGQYER
VIKSYRIFTVASMLLTLPFWIFSMAAPEMILGFMLPDQVFTGTQSMYFRIHMAILPLLSMIFMAMTFFPSINKGKPAAII
GIARQLVLYIPVMLILPKMMGVSGIYFGSLAIDAIIVLWTVVMVRKEFTRLRTKDNANLTLN
>lcl|NZ_CP026520.1_prot_WP_042228184.1_2501 [locus_tag=PC41400_RS12755] [protein=VanZ family protein] [protein_id=WP_042228184.1] [location=2763719..2764867] [gbkey=CDS]
MSAYTQPIFVAAIFFVVLTFALFIPWLIYTYRKYNFLPFSTTLISFSFIFYFLAALFLILLPLPEIRNTCSLQKPGTQHY
SLLPFQFIVDTFGNSGIVLSQPATYRLLFNQPSFYQAFFNFLLLLPFGVYLRYFFQDKKYWKKALGITFLLTLFYEVTQV
TGIYGIYNCPYRIFDVDDLLLNTTGGVLGFFIAPAVLALFPSKKKINEKAERLLALDEVRGMSVLLAVIIDLFISNMVVK
VVLSMTNVNMISEFIASTITLSVTFFIIPWIWNGSTVGTKIMRYRYDSKVSRVETNKRLFKRFGAIYAMYFITAIANTLN
NVEISMDSPFYKASIFLTLGAGLTFITLTIVLFIHIVFVVFGKEKRRFYFDEASDLYTTRKK
>lcl|NZ_CP026520.1_prot_WP_042228182.1_2502 [locus_tag=PC41400_RS12760] [protein=ABC transporter ATP-binding protein] [protein_id=WP_042228182.1] [location=2765106..2765873] [gbkey=CDS]
MEPLLKVENIKKVYGKLSAAYTALENVSFDIHAGEFVGIMGPSGAGKSTLLNLLATIDSPTEGKIYMKDKSIHDMKEAAL
ADFRRDNLGFIFQDYNLLDSLTVKENILLPLAIAKIPAKKINHLVTQISKVFGIEDLLTKYPYQISGGQKQRTAAARALV
TEPALILADEPTGALDSKSATDLLESLSTLNENNNSTIMMVTHDAFAASFCRRIIFIKDGSFSTEIYRRDQTRKAFFQEI
LDVHAAIGGEVDDVI
>lcl|NZ_CP026520.1_prot_WP_042228181.1_2503 [locus_tag=PC41400_RS12765] [protein=ABC transporter permease] [protein_id=WP_042228181.1] [location=2765860..2767779] [gbkey=CDS]
MTLYSLARRNIARNFSQYFLYIASMVFSIIIYFTFVTLKYSDTIAEQTASSQKLGSLMSGAAVILIFFVFIFIAYSNSFF
MKKRKKEVALYALLGVRNRQIGFMLFFENLLLGLFSLVVGIVLGFLCSKGFMTILIHLMGYDVVAPFTFSAAAAMNTAIV
FLVIFLVTSYQGYRLIYQFKLIDLFHASKKGEVAPKSSMIVAVLGVVLIVLGYWLALQDLFTSKVWEKVGFLMTVLIILT
TVIMGTFLLFHSVTGFVLAMIKKNEKWLWKGLHVITISQLLYRIRGNARTLTVIAVLSATTVTAGGAVYSLYYNTNDSAK
TINPNTFMYQSTDAEIDKQVSAVLSDTKYDENIEALSITFHTKELNSPSTFDSSDKRGYTVIDEKTYNKLASVQHKEKLN
VNDGCAVILDIGYDKRFSPIYTGKTITTGNGHLLTFQSFKTQTVLNGGTAGITVVVSDDQYKVLRNDGEVLHYRVLGVDR
ASVDLSKKIANQLPEEALFSSAPQDYQTSLESVGSLLFIGSFLGLVFLAATGSIIYFKVLTEAEEDKSQYNMLHKMGVSA
KEMRKSIASQVFVIFVVPLAVGLLHSVVALKAFSGLLMIDLAKPVLLWMIAYTVIYGLYYFLTVASYNRIITLNNRTEG
>lcl|NZ_CP026520.1_prot_WP_042228179.1_2504 [locus_tag=PC41400_RS12770] [protein=YxeA family protein] [protein_id=WP_042228179.1] [location=2767785..2768138] [gbkey=CDS]
MRKFLITIGILFVIGAVGVFALAKIDFNRFNAESYYLQITEDGKPHEYKLSDGTVMTSYSYTLDATNAKGKTIPLEFNAQ
KNLRKGAYLKMYVKKGDQVSSYDEVKLEDIPQKAQKN
>lcl|NZ_CP026520.1_prot_WP_042228177.1_2505 [locus_tag=PC41400_RS12775] [protein=HAMP domain-containing sensor histidine kinase] [protein_id=WP_042228177.1] [location=complement(2768212..2769654)] [gbkey=CDS]
MKWKLTLRFLRSVMTIVVIVGIVNAILFLSLFFMQLKYNIFGTEEASAETFSRTFSQYVELHSGKPVVNEEGLKKLQDMS
AWVQFLNNHGEQVSAFYAPEELQTEYSPIEIVQMYKYKELDASTTVFVSKAQNFSYFLGIKNPKVARYVMTYDYAKIFKD
VNVFLSIFFIVDVLIVIIVGFLFGKKLTAPLHTLIDGIQQLRERRFKKMPTPRGVYEDVFRNMNELSEKLDRHEEERKRL
DQMREEWISNVSHDMKTPLASIQGYAEFINDSATDLTAQELQEYTSIIKSKSMYMKDLLDDLNLTMRLRNHQLSLQFEDV
NLVQFLREMTIELLNDLALGERQVEFEANVVKAVHKVDKKLLKRAILNFIYNALVHNDENVVVKIRVETIHLKYGQEHPS
GSLSPGPEFHTQISISDNGHGIPAKDLEQVFERYYRGTNTASKHGTGLGMAIARDIIWAHEGKLDLKSVEHIGTTITILL
>lcl|NZ_CP026520.1_prot_WP_042228175.1_2506 [locus_tag=PC41400_RS12780] [protein=response regulator transcription factor] [protein_id=WP_042228175.1] [location=complement(2769661..2770362)] [gbkey=CDS]
MVSRKWVLIIDDEDDLARLMKTVLHKEGIADVVTAGTVAEGWEKFQEYNPELILLDIMLPDGEGYDLCRKIRETSNAPIL
FMSAKDEEINKLLGLAIGGDDYITKPFSPKEVAYRVKAQLRRAGFSEEKKAPKSLQVGPFELSANETELKKNGEAVVLTA
KEVGLMSCFMHHPNQIISKETLFGQVWGEDFFGSDNTVMVHIRRLREKIEVDPSKPSFLLTVKGLGYKLHTKG
>lcl|NZ_CP026520.1_prot_WP_042228173.1_2507 [locus_tag=PC41400_RS12785] [protein=glycosyl hydrolase family 18 protein] [protein_id=WP_042228173.1] [location=complement(2770709..2772610)] [gbkey=CDS]
MKAVRWILTGAVVVSLCSPLQAQAADKTTKYRVYQNNKIVTELSDYSQAEAYAKQFKNSHVEEISTRKWLWNNFPRYQVF
QYDFTQPQWTFPTLDAAVAEARRWGHASVRDLQSGGWVWNNYPKYRVYQGDNTQSLWDFTTLADATAEARKWRHAHIIDL
STNKWVWDNISAADKSSLRSGTPVYRVYQDTYSAPSWSFAYLEDAVNESLKWGGSKVVRTTDNAVLHTNEASYKVYQNDN
FLSEFQSLDAAATYASRWGHSEISYNGKIIWSNYPYYQVLQNNTLIGDFNTIPQALEYASKYSNSTIRTLDRNVIWDNLR
KMQYWGWTGSSAASTIQAQASGTVGLDVVSPTYFQLADASGKVDDKSEAATVDWLKKQGFTVYPLINNQFNAEMTSAFLA
NPAAQTAFIRQIVDKAAALKVDGLNLDFESLSAKDRAGYTTFVKQFADAAHAKKLSVSVDLPRGSIKWNDKTAYDHEKLA
SYVDYIAIMAYDQYWKGSTEPGSVSGLPWAEQGVIEFLSYGIPRDKLIMGIPYYVRDWTLDAAGKPVENKAILLKDIPAL
ITAKKAVKTWDNEFKQYRVDYTEDGKKHVFWLEDNDTVKQRLAIARKYDLAGVAAWRLGYDTPQLWEMMLQEK
>lcl|NZ_CP026520.1_prot_WP_009674362.1_2508 [locus_tag=PC41400_RS12790] [protein=DUF2653 family protein] [protein_id=WP_009674362.1] [location=complement(2772745..2773020)] [gbkey=CDS]
MKLTEQEITNALCVNMAQRKQLHPSQVEVVLMWDEDLGFSAEVFAEGRNQILIEANIYEAIMQYLSQQQIRVFRDQIQLD
LEDEIVAYIHE
>lcl|NZ_CP026520.1_prot_WP_174446790.1_2509 [locus_tag=PC41400_RS12795] [protein=hypothetical protein] [protein_id=WP_174446790.1] [location=complement(2773017..2773448)] [gbkey=CDS]
MKKWLLLVMACMLTFSVTMPGTVEAKKSYSSTKKSYTPSTTNKSNDTNVNKSTPSNSSKTTSTAPTQKRGFFSGGLMKGL
LIGGLAGMLFGGLFGGMGMLGNVLGFMVNMLAILLIVMVIRRIVVYFMDKRKADKMKEKHNLT
>lcl|NZ_CP026520.1_prot_WP_009674350.1_2510 [locus_tag=PC41400_RS12800] [protein=DUF1540 domain-containing protein] [protein_id=WP_009674350.1] [location=complement(2773660..2773821)] [gbkey=CDS]
MAKDVLCEVNSCRHWSNGNKCAAESIYVVSHHDGNQARHSEETDCKTFESKLS
>lcl|NZ_CP026520.1_prot_WP_241688856.1_2511 [locus_tag=PC41400_RS12805] [protein=DUF309 domain-containing protein] [protein_id=WP_241688856.1] [location=2774018..2774596] [gbkey=CDS]
MSAIIWDEGKRTMERYPKEVLNYLVHFHGDRDFFECHEILEEYWKENPGHPYAETLVGLIQVAVGLYHQRRGNKAGAVKM
LGSSLSYLKEEDLKCLGLAGEDLVARVRQRVREAELGAVPYEDLDLPVEDPSLLEACKRECALRGFGWGEASDASNLMLV
NRHKLRDRSDVVEERARQLEARKLRRKGGEGQ
>lcl|NZ_CP026520.1_prot_WP_042228171.1_2512 [locus_tag=PC41400_RS12810] [protein=response regulator transcription factor] [protein_id=WP_042228171.1] [location=2774593..2775321] [gbkey=CDS]
MKDKIAVVDDEPYIAEVARLYLEHAGYQPLLVDHAAEVLPLLREHSPSLVILDVMLPDGSGFDVCRSIRSMNKPLCDVPV
IMLTARGEAFDKLEGFNVGADDYLVKPFDPSELIARVKAVLRRSKPEEAAAASADPSLPQPIRIGALTIDFARYRVELEG
SRIDLTPKETELLYFLASKAGRVFSRDDLLAGVWAFDFPGGTRTVDAHVKNLRKKLGSHPEWRIETLWGIGYSFEVTAHE
KG
>lcl|NZ_CP026520.1_prot_WP_042228169.1_2513 [locus_tag=PC41400_RS12815] [protein=HAMP domain-containing sensor histidine kinase] [protein_id=WP_042228169.1] [location=2775308..2776837] [gbkey=CDS]
MRRAERLLEFWRGRFGSLYLRVFLLFLAVCVLFFVGLALFWNMYFQNFFYKDKQEFLKKGYVEVTKLLAPFQEGTLSTRE
LRMATRIVARGVNGRIFLVDGKGVILNGSTEYETFKLPPTLEKSLGSGLEGKSGFLFDQPPSERHPGDKRPMDTLLTYYA
PAQIGGETIVVFLQVPVAEIRQTISVIRLIILLPLLFSLLAVGLLLYFISRRMTAPLKQMNELATAIEQGDLSLRVPVTS
KDEVGELAESFNRVVDRLQEWEGTRQEFLAQVSHELRSPLTSLRGLISAMRDGIIEPDKYGHYFRICESEVRRLERLVQE
LLDLARIQNGAGLRELKPVDMVAQTGEVLDVMREIFARKRLELSERVDVPAGEGLFASLDADRYAQIVHNLLYNAVQFTP
EGGAVKVRLERKADKAVLTVSDTGIGMTAEEIQRIWEKFHKVDRTRGGTTEGVGLGLTIVKHLVEAMNGRITVQSRPGEG
TSFYVEFARVNGNGGYPGGIPGNPDSSAG
>lcl|NZ_CP026520.1_prot_WP_042228167.1_2514 [locus_tag=PC41400_RS12820] [protein=hypothetical protein] [protein_id=WP_042228167.1] [location=2777002..2777691] [gbkey=CDS]
MSFRRKLHLRWIGAACLLPLACTLTFAFAGSMSPHAGTLEPEAEALSASQPAYETNGAAAFQGGNRPENMEPAIQTSRER
LLVLFCATLLEVGKEAGFSLTPAQAEQLAGILQRSVGRGEMTQEESEEAKHLLTGKQRDMFEKVFADDAKRSLQGRALET
GGSREHCGDTAPNFYGSVPNGGTDAAEESSALRPAACGPLKEEPELNLEQQTLHMLNRQKSAAITGRNK
>lcl|NZ_CP026520.1_prot_WP_042228166.1_2515 [locus_tag=PC41400_RS12825] [protein=HlyD family efflux transporter periplasmic adaptor subunit] [protein_id=WP_042228166.1] [location=2777695..2779404] [gbkey=CDS]
MKWSGKVIGMSLAGVLLVGGGGYFAFGAVKKPMAGAAAQEVVTAVKKGDIRSTVSGTAQFEPKEKQTIVSPADATIKAMH
LTRNMPVKAGDVLVELSSPTLENNLQKALMTQADIEKQIAELQRQQANMGTSAPIGGQLVLAPNIEVGSNVNKNTKIATV
NDTTNLLATVLFPYEEALQLRSGEEIDLAVDGFSITKTGKVDAVNKTAKSDGKGGKALEVQIRIQNDGTLDAGLKVKASA
RIGNVSVDAVAESSLEYAKSAPVVAGVQGTVSSLPNKSGTLVKAGASVAVLENDTIQSDISDKEAALEQQKLAVQEARDK
VADLVVKAPFDGVFSTDFVNTKENILASLTPGAKVKNAFEFGAVANMELMQLPIQVDELDLNAIKTGMKAEVSVDSLQGK
KFEGEVTQISSMGTTTNGVTFYDVILAVKNTNELKYGMTATANILIENKTGVLYVSPEALQSRQGKRYVTLKKADGTLEA
QHEVKIGIRSTTQVEITEGLKEGDKVVVPQKAQPTKMTQQEIDRLRQQYQGGQGGQGGGGFQGGGGLPGGGQGGQGSQGG
GGQRANQTR
>lcl|NZ_CP026520.1_prot_WP_042228165.1_2516 [locus_tag=PC41400_RS12830] [protein=ABC transporter ATP-binding protein] [protein_id=WP_042228165.1] [location=2779432..2780142] [gbkey=CDS]
MNMIELQGITKVYRQGAQELTILKKLDLSVAAGEFVAIVGPSGSGKSTLMNTLGLLDVPTTGSYFLDGVRTERLSDNQLS
ELRNRKIGFIFQQFNLLPRLSAFENVELPMIYAGIPAKQRAEKARAVLEKLGMKGREHHKPSELSGGQQQRVAIARSLTL
SPPLLLADEPTGALDTRTGQEVLELILELNGQGHTIALITHDLHIADCAKRIVSIRDGEIVGDRLSLPQAAQGVPS
>lcl|NZ_CP026520.1_prot_WP_042228164.1_2517 [locus_tag=PC41400_RS12835] [protein=ABC transporter permease] [protein_id=WP_042228164.1] [location=2780139..2781317] [gbkey=CDS]
MKFTELIRMAMKTVTGSPLRTFLTMLGVIIGVCSVIALVALGNGTTDKIKAQYENLGTNLLVVNVNGMGRATQLDYNELM
QFEGFPEISAIAPTMTKNGAHIKADRTQDTSNVIGTNDRYQSIVKADVASGRFLSPSDLEFRSKVAVIGSEVAKTYFGTL
DPVGETIRIEGIPFTVIGQLKEKGSNIGGTSVDSSILVPLETARRQFQLGTIRTTYIEAPTRDDIYKAQNTMKSYLTYKF
KSDTGFTLINQDELLAASTKATDTLTKQLVSVAAISLLVGGIGIMNIMLVTVSERTREIGIRKSIGAKRRSILLQFLVEA
TVISGLGGLFGLLLGIGIVYGWPLFSKEQTLRLSLDVCVYAFAFSVLVGVVFGLYPANKASKLHPIDALRSD
>lcl|NZ_CP026520.1_prot_WP_042228162.1_2518 [locus_tag=PC41400_RS12840] [protein=hypothetical protein] [protein_id=WP_042228162.1] [location=2781350..2783629] [gbkey=CDS]
MEKDKRNKLPKLLVTLTAAVAISSAGWSGPFTGKASAGAAGIFVSPKVYFTVEDVSLAKGTGKSSLHFTLLLHNEGSGTV
DLNAFAVSVNAPDGSRYTPKLSGTSSARVWPDSTGRYIYTAEVSSQASLDTLKLSLNRWASDGISKGADIGSLPLAADAA
PAPDKPVRVQLNEADASLPEDAVLTIRAKDSIRWNRDSGQELYTHLTVVNQGSAALKWNDNVQLSVQDAKGRTYTAAVVS
GADILLEPQEPAVLTVKTSAAGSTSPGTDLKLVLGAKNSGGAFTIGTVPVPGETTAMGVGDAIPYVYGDGLELTYRSAVV
NAKTDGTTVQTVVEFRNTAKEYVRLPDLSAVYQFGSRATVDGVIDAKPSSAFLAPGESASYRYTASLPAGVSKDAIELLL
LAKESLAETAAESGGGSTGSAAASGTPGTSASGAKASASSAGTAAGGAASSAAKATASADSSSAAAASSGSGKTTAAGST
GYVPAASLSLAKAESGKGGDYAAAQTYTLGEKLPLPNGALEAGLGVSLVEFHKHENEEYGYQTAVAKYKFTNTSGKTLDL
PQLGMELTTPDGLVYAGTRQSSAAASILPGTSAVVSYSYMVPPSDKTETLAMTVTDANKMSLGSYRVAFQAEAENGSVSF
YPFEVAFNDSTVGWTYSTKTDAQYTYKLILDMTITRKEQVIVDSNFSKMEIELLDPDNRSLASQTVPFLGTGRLVSGKQT
FFFNDVKSEQFSSGAHFNIYEVIETPNGPVRRLIKTIQG
>lcl|NZ_CP026520.1_prot_WP_042228160.1_2519 [locus_tag=PC41400_RS12845] [protein=MFS transporter] [protein_id=WP_042228160.1] [location=2783889..2785085] [gbkey=CDS]
MKRFAQASYVMYFLAGLVITTIGSVLPQILRHYEVTYTMGGQLVFWGSIGFLVGVPVATLLLNRFSEKQILALACLIIAG
SQFMMWSLPPFSALLVINFMNCLGTAAVETIVATLMMEVFVGRRAVVMSYLEVSFGLGALLMPVISSLLISLDLWRYSFV
VTGVMALAMTGVWVVITYSKTDLDENAAKDASNMAPPENISARLKWLLLVFFGFMIFMYCGIEGSMNNFLSSIFISYLDA
APFLAALSVGLFWGAMVLGRLATGWIIRKVTYSRFLFISMASSLIVLAAFIVLQNIWTGYILIIALGLLLSGVYSITMVY
ANHTFPGLARLVTSLITGLAGLGGAVFPALIGYVMDQAGSSWALWMTVGFAVFYLLGLAVVEAVYKKTRRSGNPAQAS
>lcl|NZ_CP026520.1_prot_WP_042228158.1_2520 [locus_tag=PC41400_RS12850] [protein=carbonic anhydrase] [protein_id=WP_042228158.1] [location=complement(2785226..2785828)] [gbkey=CDS]
MSLISEILNYNEAFVDSKEYEQYLTTKFPDKRMVILTCMDTRLVELLPKAMNLRNGDSKIIKNAGALVSHPFGSIMRSII
VAVYELDADEVLVIGHRDCGMTGLNSEKVLSKARERGVQNEVLDTLHNSGINLDKWLTGFEQIDEAVIGSVNIIRNHPLL
PKNLPVHGLVMNPQTGKLDLLTEGYDATPVKTNPEEFPFI
>lcl|NZ_CP026520.1_prot_WP_042228156.1_2521 [locus_tag=PC41400_RS12855] [protein=iron-sulfur cluster biosynthesis family protein] [protein_id=WP_042228156.1] [location=complement(2785894..2786217)] [gbkey=CDS]
MYIHFSEAAVEALTNRLPEDAILRLFYDTEGCGCALAGVAALHIQAAHTEGDVEAQSNAFTVQYDPRKEVFFEEKLSVDY
MPQRGTYVLKSSQQTYNPAMRLVNCVN
>lcl|NZ_CP026520.1_prot_WP_042228154.1_2522 [locus_tag=PC41400_RS12860] [protein=carboxypeptidase M32] [protein_id=WP_042228154.1] [location=2786329..2787849] [gbkey=CDS]
MSQTDYQGKLDEFRKYIRKIKHYEEAVGLIYWDLRTGAPKKGASERSEVIGELSTEAFRLSVSDQMGGYLDFFSDAARIE
ELDETNRRIVQEAKKEFDRSKKIPPEKYQDYVVLTSKAESAWEEAKEKNDFAMFQPYLEKIVETNLEFVELWGYKGHKYN
TLLDMYEPGMTVEKLDEVFGALRSEVVPLLAEIKKAPQPDRSFLRQTFDKEKQKHFSLFILKELGYDFEAGRLDETVHPF
ATGLNPGDVRITTRYLPEDATSALFGTIHECGHALYEQNISEDLIGTNLCTGTSMGIHESQSRFWENMVGRSREFWSRYY
ADLQRTFPGQFDNVSEEAFYRAINHTEPSLIRIEADELTYNLHIMIRYELEKALFSGELKVADLPEAWNAKYREYLGIEA
DTNAEGVLQDVHWAGGSFGYFPSYALGNMYAAQFTNTLRKEIPAFDELLAKGDFAPIKQFLVENIYRYGKSQTPNEIVRR
VTGEDLNPQYLVDYLKDKYKPLYGLS
>lcl|NZ_CP026520.1_prot_WP_042228153.1_2523 [locus_tag=PC41400_RS12865] [protein=hypothetical protein] [protein_id=WP_042228153.1] [location=2788072..2788512] [gbkey=CDS]
MKLFKGSFMLLISLIAIISLNLQTVSAESVAWYDGLPAKAGQWTPVHSDHIYSEPFNVTGVSSVTIKGYQYFPADSTANV
KYQIVLRNAIWDEVLAETTVTGDYPKEGTWFSRVLTLNKPIIGSVSLKIINLNPINGVKVAGNVYK
>lcl|NZ_CP026520.1_prot_WP_042228152.1_2524 [locus_tag=PC41400_RS12870] [protein=ABC transporter substrate-binding protein] [protein_id=WP_042228152.1] [location=2788645..2789652] [gbkey=CDS]
MKRRSIVFLLLSVLLAGLVLPGCSKQEEVSKIRIGEVTRSIFYAPQYVALSQGFFKDEGLEVDLTTTAGGDKTMTTLLSG
GIDVALVGSETSIYVHQQGSDDPVINFAQLTQTDGTFLVSRTKPAGDFDWNSLQGKTFLGQRKGGMPQMAGEFALRKHGI
DPHKDLNLIQNVDFANIASAFSSGTGEYVQLFEPQASVFEKEGRGYVLASFGVESGKLPYTVYMTKQSFIKKNPNAVQKF
TNAVQRAQNWMHTKSINEIADAVLPYFKDTDPEIVRSVVKRYKDQNTYSADPIIDEAEWNNLQDVMTQAGELKAKADWSK
LVDNSFAEKAKVNAK
>lcl|NZ_CP026520.1_prot_WP_042228151.1_2525 [locus_tag=PC41400_RS12875] [protein=ABC transporter ATP-binding protein] [protein_id=WP_042228151.1] [location=2789737..2790528] [gbkey=CDS]
MDTAATVQVSGMTHVYVSDAQANPALENIGLTIRPGEFVSLLGPSGCGKTTLLSIIAGLIRPTRGQVLINGREIVSPTSR
VGYMLQQDYLFPWRTIYGNAVIGLELLGRLNKETGAYVDHLLEEMGLAAFRDYRPAQLSGGMRQRVALVRTLAAKPELLL
LDEPFSALDYQTKLQLEDLVVQTLRSRGKTALLVTHDITEAIAMSDRIILLEPHPGRIRREIAVPEHIRRTLPVEAREKP
GFHELFHEIWQEFERMGGKEAET
>lcl|NZ_CP026520.1_prot_WP_042228150.1_2526 [locus_tag=PC41400_RS12880] [protein=ABC transporter permease] [protein_id=WP_042228150.1] [location=2790525..2791409] [gbkey=CDS]
MNHYPADETAQNQEAGTRAQVHGKEHTDGRTLADTIHAAHLETKRKVAVRVRVTQLLILVIGLGVWEAAGRLKWIDVLLF
SYPSKVFVLLASKIADGSIFPHTGYTVYETVVGFVLGTLFGTALAMLLWWSPFLSRVLDPYIVVLNSMPKVALGPLFIVG
FGPGMLSIIATTLSVTVIITTLVVYGSFKEVDDNYVKVVRIFGGSRGEVFRKVILPASFPTIVATLKANVGLAWVGVIVG
EFLVSKMGLGYLIVYGFQVFNFTLVIASLVVIAIVATFMYQAVVYLENKLMSRR
>lcl|NZ_CP026520.1_prot_WP_042228148.1_2527 [locus_tag=PC41400_RS12885] [protein=aromatic acid exporter family protein] [protein_id=WP_042228148.1] [location=2791519..2792505] [gbkey=CDS]
MNIPFIGIRVLKTALAVIVSITLARWIGLTSPNSAGLLAILGIQVTKRRGINNAFQRIAASILAVLFASVLFSLLGFRVW
VLSLFVLIVFPVLHRIKLSDGAVTASVTIFHIYNAQAISAPLILNEIILLIVGLGSATLINISYMPKADKEIEFYRSKVE
GLFSDVFLHIAEHLRDTTVVWNGKELLELDEVLKTGTETASRSLENALLFGTHTYTDMQLYFYMRNEQFESLQRMALLVS
QVSRELPLGESLASIFEELSKDCKVDYYTGRSEKKLAALQANFKIETLPQTLSEFEARAALQQLTRELSHYLNIAKKQKK
QKPASARS
>lcl|NZ_CP026520.1_prot_WP_042228146.1_2528 [locus_tag=PC41400_RS12890] [protein=hypothetical protein] [protein_id=WP_042228146.1] [location=complement(2792573..2793229)] [gbkey=CDS]
MNCPHRKRCHKKTIIVCKPKRKKKVVIVKKKIVKVSCPPPTVNVTPIPGPQGPQGLPGAQGPQGPAGGISSFAFLCSTEA
QTIAAAPTPGGQGGAVSFNGSVISATAITFAPPSSIVINENGFYKITYEVFPTQGNNAFGLFFDPDAAGPAPATLVLCSN
YGTGAGNNPYPGQVIVQLTAGGVLTLNRIDNTGTLTLQNAIGGGTPTVSASLSIEKLA
>lcl|NZ_CP026520.1_prot_WP_042228144.1_2529 [locus_tag=PC41400_RS12895] [protein=outer spore coat protein CotE] [protein_id=WP_042228144.1] [location=2794284..2794847] [gbkey=CDS]
MSIDRNLQSREIIVKAVCGKGRKFSHVSHTVTPPHAPTSILGAWIINNQYEAVKSGEGVEVVGTYDINIWYSYDRNTKTD
VAKETVSYAEVVGLSYLDKRHKSVTAEVSAVATQEPNCVEASISGGGNVLIRVEREFHVELTAETKVHVIVDPSGGDYEE
KHVDFDLIGDDGDFEDLNPDLLEDDLS
>lcl|NZ_CP026520.1_prot_WP_042228143.1_2530 [locus_tag=PC41400_RS12900] [protein=hypothetical protein] [protein_id=WP_042228143.1] [location=2794949..2796451] [gbkey=CDS]
MQLFVQYSGRDKQTASVISRELDLLNIPHGEKLPDKTPVRIAAWGKLSAEQERVLTALPRRVLLQPPEAVSLALREEDAR
MQLAVHGIRTVRNFPGEAADRRARGSRPGGPYSYEYRIPVFHMEALAVYRQKNNLLLLPGTAGKKPSGFQEVSLDQQTFH
TQRAVRESVKAVYALGLDYGLVHIAIEPSGRTLVLRVDPSPPLDEPLAALFAQAIGRYADQLSQELARTEPAVLGSDPEF
LLVNPRGKVVSASKYLEREGQVGCDAIVLSGHRVILPLAELRPQPSSEPAQMAANLRRTMRMAARKIGDDSLAWIAGGMP
LPGFPLGGHVHFSCCWLNSHLLRALDNYLALPLMMIEGETTAQRRPRYGLLGDYRRQRHGGFEYRTLPSWLVSPEIALGV
FALARIIADRYFDLDQRPLDDMEAQKNYYAGNKPALADAAEGLWTDLEKQPMYGEFARYLRPLREQSRRMEGWQELADFR
RSWKIGPYGRKQDAVQENML
>lcl|NZ_CP026520.1_prot_WP_042228141.1_2531 [gene=mutS] [locus_tag=PC41400_RS12905] [protein=DNA mismatch repair protein MutS] [protein_id=WP_042228141.1] [location=2796504..2799227] [gbkey=CDS]
MTQYTPMIQQYLAIKAQVPDAFLFFRLGDFYELFFEDAINASRELEITLTGRGGGTEERIPMCGVPYHSADNYISRLIEK
GYKAAICEQVEDPSVAKGVVRREIVRIVTPGTVMDGKTLGDASNNYIAAVAQVGGRYGFSACDVSTGELYATSLSDSLQM
LIDELNAYAPAEIIGDGPLLDRVRGQIPLTAKSTVFTAWERAADDKLAPQFGEEAAGKLDASARRVVSLLLAYLHETQKQ
SLTHIKEIRAYSPQQFMTMDPFTRRNLELVETVRERSRKGSLLWLLDETVTAMGARMLRRWIEKPLMQADAIRSRHDAVE
TLYKQFILRDDLRTVLKEVYDLERLVARMAYGSANARDMIALRVSLEQVPRLKNVCAGAGARVLERLVDEMDPCSDISEM
ILGAIEDDPPVSVRDGGMIREGYHDYLDQLREASTNGKKWIAELERSEREATGIKSLKVGYNKVFGYYIEVTKANVGSLP
EGRYERKQTLANAERFVTPELKEKEALILEAEDKRTDLEYELFVQLRERVSAQIGRLQKLAEAIAELDVYQSLAAVSASN
RFIRPEIGDGYDLHIEEGRHPVVEAVLESGAFIENETKLTAGDGSMLLITGPNMAGKSTYMRQVAVICLMAQIGCFVPAS
SARIPIIDRIFTRIGAADDLIGGQSTFMVEMMDIQVMTEKATERSLVIIDELGRGTSTGEGMAIAQAVIEFLHHEIGCKT
LVSTHFHELAHLEESLPQLRNYCMAVKESGQQVTFLRKLIRGAASTSYGIYCAQIAGLPDAIISRSYELLQTFEARAAAD
GPEPGAGSSGGAVLTEEPAARTSANAAGAGEANAPYGVPDGALSQRSAVEQLSFFEEAKPVRAQAPSAKPDPQTQAVIDE
LKALDLLNITPLQAINVIYELKKKLQA
>lcl|NZ_CP026520.1_prot_WP_042228140.1_2532 [locus_tag=PC41400_RS12910] [protein=S41 family peptidase] [protein_id=WP_042228140.1] [location=2799359..2800798] [gbkey=CDS]
MGFPGIRKLKTAVAAIVVFGAVIPASAWAQESARTTDEVRQVLGQYHVSGITTDELKGLSIADMLKKINAKDPYTTYFTK
EQLEKFTDSLENNYYGIGAQMGYDETGVYADKIFEGSPAEKGGLKKGDYILKVAGVSTQGKNLSEVVSGVLGPEGTKIEL
TVKRDGKELNLTLTRAKIHLPNVEGYLITPSTGYIKIDSFSSDADEIFDATFSALQKMGMKSLVIDLRDNPGGLVESAQN
IASNFIKEGVFAHTKNRNNEDTPLQIKGGSTQNIPVFMLLNENSASASEMLAGALRDYGVAEVIGMKSYGKGSMQSLHSL
SDESAVKVTFEEYLTPKMKPVNGIGIAPDIESDGYTAQLITALRKAGTNSFEVRVTKHDVTVNQVEVTEPYRPIREGGNV
FVASRTLAALIGADIKWNQAEQAVNISTGKGIHTFAMSPDTLLLKNGTSYINLDKFDDIYPQLHWEDADNLLQLETEKE
>lcl|NZ_CP026520.1_prot_WP_042228138.1_2533 [gene=mutL] [locus_tag=PC41400_RS12915] [protein=DNA mismatch repair endonuclease MutL] [protein_id=WP_042228138.1] [location=2800805..2803078] [gbkey=CDS]
MGKIRLLDEHIANQIAAGEVVERPASVVKELVENSIDAGSSRIDVVAEEGGLQLIRVTDNGSGIESEDCELAFYRHATSK
IQSGKDLFSIRTLGFRGEALPSIAAVSKTEFVTSTDGSGLGRKLAVEGGDVVTQEETSARKGTDISVKELFYNTPARLKY
MKTIQTELGHISDYMYRLALAHPEIAFTLKHNGNSLLQTLGGGDLLQVIAGVYGSAVGKQMIAVGGESLDYKISGYISRP
ELNRANRSAMSIMVNGRYVRSFAVQHAIMNGFHTLLPINRFPVVVLHIEMDPSLVDVNVHPSKLEVRFSKEPELVKMIED
MIKEALGRQVLIPKGVQASARIKESFVQETMTLYRPEADPLLKQGITPAQWREQLSVPEDGQAPASADNREGAGEGRDTA
QEADERRRSLASLDSLAPPESGAGSAGSAAEALAPDGDGRREQQPLAAGGAAGLSGRGAADDRRTGSASDAAVPDGVPAP
AAQEAAASRLREPQSRPAAAEPYGAPYGSGAYGSQPRSGAAAGAGGRTGRAPAPDDRPYSREAAGSLLAALPTRSDKEAL
KRPDFPSLSPIGQLHGTYLLAQNEAGFYMIDQHAAHERINYEYYYTKFGEPQEASQELLVPIPLEFTSADFQALRDKLPL
LEQVGVYLEAFGGNTYLVRAYPHWFPAGDEQAIVEEMCEWILTERKGVDLSKLREKSSTLCSCKASIKANQSLSTAEMEA
LLDRLGGCGNPYTCPHGRPIVISFSTYELEKMFKRVM
>lcl|NZ_CP026520.1_prot_WP_042228137.1_2534 [locus_tag=PC41400_RS12920] [protein=class I SAM-dependent methyltransferase] [protein_id=WP_042228137.1] [location=2803282..2804055] [gbkey=CDS]
MLVTTSYDPASEDVDKAIAAARTLGGRFVPRKRLTLSGLRSRFGENEILLITKEGLKYDFGGPHALFFHPSMALVRYKRL
RQGSSDPLLELSGAGRGDAVVDCTAGLASDSILFSFAVGEEGSVTAWEHGRVLAYLLEQGLGSYRTGEAEFDAAMKRVAI
RQGDHLDGLRAMEDNSCDIVYFDPMFRAPIQESSGLSPLRSAADDRPLEAAAVEEARRVARRCVILKEGRQSGEFERLGF
RETVPTGSKLAYGVIRC
>lcl|NZ_CP026520.1_prot_WP_042228136.1_2535 [gene=miaA] [locus_tag=PC41400_RS12925] [protein=tRNA (adenosine(37)-N6)-dimethylallyltransferase MiaA] [protein_id=WP_042228136.1] [location=2804049..2805023] [gbkey=CDS]
MLNSQGAKPKLLVLIGPTAVGKTKLSLEIAERWNCEILSGDSMQVYRGMDIGTAKATPEERAAIPHHMIDIHDPGYPYSV
AEFQQRVRQLVPEITERGKLPFLVGGTGLYVESVCYEYQFLEAGADEEFRHEKNRFAEEHGDEALHRTLAEVDPDSALKL
HPNDRRRVIRALEVYHLTGKPLSAHNAAQKKESPYELCIVGLTMDRQLLYGRIEERIDIMMAEGLVDEVKSLLESGVPRQ
SVAMQGLGYKEMVSYLDGEISLERAVELLKRDTRHFAKRQLSWFRHMKDIQWIDVTDTANFSGHLEHFHAIITGKFVHDN
EYNA
>lcl|NZ_CP026520.1_prot_WP_009674373.1_2536 [gene=hfq] [locus_tag=PC41400_RS12930] [protein=RNA chaperone Hfq] [protein_id=WP_009674373.1] [location=2805080..2805325] [gbkey=CDS]
MKQSINIQDNFLNQLRKESIPVTVYLTNGFQIRGLIRAFDNFTIIIDSDGRQQMVYKHAISTFTPQRSVSLMPAPEQSAA
E
>lcl|NZ_CP026520.1_prot_WP_042228133.1_2537 [locus_tag=PC41400_RS12935] [protein=penicillin-binding protein 1A] [protein_id=WP_042228133.1] [location=2805478..2808057] [gbkey=CDS]
MAASSKTPSKGKKPKKKGSAAKVILGVIIAALLALICAMSIYLVVYLNGLKILTANIDKLDDALGASVIYDADKQEIAKV
IKENRENVKFDQFPKQLVDAFVATEDRRFFEHAGVDFWSLGRAVVKDVIARSAVEGGSTITQQLAKNVFLNSDKTFFRKG
TEMSIALAIDNKYTKNEILEKYLNRIYFGSQAYGVKAAAKRYFGVIELKDLQLWQIATLAGLPKAPSKYSPISDPEASKD
RRAVVLKLMADQGLITEEERAQAAAVDYDPKTQTPVAQSNLFPSFRDYVIKEAYDLYGVTDEQISNGGYKIYTTLNSTAQ
KAMDKAYTNAKLFQPDGPEQPMQSSMVIIDNASGGIVAMTGGRDYKSGTWNRAYKPRQPGSSFKPLVVYAPAIEKKGYLP
TSMLDDTKQEFGKYAPRNYDGQYLGQVDMTYAIKKSKNIPAVWTLSKIGINTGKDFAKQLGITFDENDNNLAMALGGMTK
GESPLQMAQAYAAFANNGVYNQGHAILKIEKSDGTIIDYKAEPKKVMSEKTSWYMTQMMKEVVTSGGTGVRADFGRPLAG
KTGSTQLDLKGLESHYRDIWFVGYTPQWTGAVWAGFDKTDSKHYIRKNYSDRANMVFSAVMAEAMKGIPVKDFERPSGGK
QIEQEKPPVQEKKSVTDLRAGWQNSGVRLSWTAPEGTSAYQVFRKGPGETDFSQLTPSNSTEYMDQTVQPGGTYEYYVVP
ANGDAAGVQSNVASVNVPGTEPTPTPPNPDDNNGENGNGNNNGNGNGNGTNNGNNNGNNNGNGNGHNNGNGNGNGNPGKD
KDKDKDKDKEKNPGNTGGGTGNGGSTGGTASPTPTPTPSEPAGSGTTGSRDGELQPNNG
>lcl|NZ_CP026520.1_prot_WP_042228131.1_2538 [locus_tag=PC41400_RS12940] [protein=hypothetical protein] [protein_id=WP_042228131.1] [location=2808206..2808541] [gbkey=CDS]
MRTPSVQEWSRKWVDDHLDLYNYAGSLGDTEWQQAIIELLKFRDNYVQQEIRARLRQDLWKMYDSINLKMLELFEQLRQS
KDNTQIETIREQVWVLKSQRIEISKKILAVK
>lcl|NZ_CP026520.1_prot_WP_009674360.1_2539 [locus_tag=PC41400_RS12945] [protein=hypothetical protein] [protein_id=WP_009674360.1] [location=complement(2808687..2808908)] [gbkey=CDS]
MEKMCMQMMMTMCMDEMMCKMNTMKSMMDSMMQMNMMSEMNMDMDMMMGKMQECDEKMTMMMKMMSEMKQMSM
>lcl|NZ_CP026520.1_prot_PC41400_RS12950_2540 [locus_tag=PC41400_RS12950] [protein=SDR family oxidoreductase] [pseudo=true] [location=complement(2809126..2809994)] [gbkey=CDS]
MTTNNEQKQTFPPQHQNNQPGIETQMTPQPQFEDSNYRPAGKLKGKVALITGGAAESGVRLPWHTPKKGRTWPSFT*ANI
RMRRKPIPL*SGRAANAF*FPATSETNPSVKR*STRLSASSANSTFSSTTRRSSIRRTALKILRRNSSKKRSGPTYSRCS
SLRRPLSLI*KEEAP*STRLPSPLIKETRR*STTRPPKGLSFPLPARSPNPLSKKGSA*TA*LRDRSGHRLFRPPLRRTR
*RLSVRMHL*KEPGSPRSSLRATSSLLRTTLPTCRGKFFTSTEAPSLMA
>lcl|NZ_CP026520.1_prot_WP_042228130.1_2541 [locus_tag=PC41400_RS12955] [protein=GNAT family N-acetyltransferase] [protein_id=WP_042228130.1] [location=complement(2810042..2810815)] [gbkey=CDS]
MNTKMDWLALEKWAADQWPPEESEALGHWLLRASGGFTKRANSVLAVGDFPAEPDWYERICRFYRDRGQPAVFHISNSSP
PELDLMLEDRGGIVTVPCFGMIAPAAELNARLRPHQDRTDVSAELLASPDSAWMEDYMLLEGHAVEARSFYEGLLERMPA
PKAFVRLLSRDGETVALGTAHASGGRAIWSNIIVREDRRGHGFAKALLAHLAAWCADNGARSVFLQVLQDNAAALHLYES
AGFKPEFRYHYRVMQQT
>lcl|NZ_CP026520.1_prot_WP_042228128.1_2542 [locus_tag=PC41400_RS12960] [protein=lipoate--protein ligase] [protein_id=WP_042228128.1] [location=2811026..2812036] [gbkey=CDS]
MLFIDNQSITKPQINLAIEEYALKNLPADETYLLFYINEPSIIIGKNQNTVEEINTEYVEQNGIHVVRRLSGGGAVYHDH
GNLNFSFITKDDGESFHNFRKFTEPVVQALRKLGVDAELSGRNDIQVGERKISGNAMFATKGRLFSHGTLLFQSDIEHVV
SALRVKADKIQSKGIKSIRSRVANIAEFLNKPMTIEEFRQELLETIFGMSASEVPQYVLTEKDWKAIHELSAERYGNWDW
NYGKSPKSELVNSKRFPVGTIDVRMNLEEGRIGKMKIYGDFFGVKDVEEIERLLTGVKYDTQSLEQALEGTDITAYFGAV
GKEEFIGLLTLKGEEA
>lcl|NZ_CP026520.1_prot_WP_042228127.1_2543 [locus_tag=PC41400_RS12965] [protein=hypothetical protein] [protein_id=WP_042228127.1] [location=2812036..2812944] [gbkey=CDS]
MPRLSRKSSPPAACPQCSGKKYISFNWARFQHGHDAESKQVQRALKIERQLKHGWEYACENCGSRWYLGRDGIMMSYIHP
SKAGLVEHWNAGTYEVSDEIWDELGKIGATAQGSIAGGRTSAEVPCRVTTVRGETFDMAYVSFQSRPPIDLWQDGRRIRF
CDEVASVEPSPYTLSADVRLATSRAQEIRMGYAPTAVQAPDGHLFQLNWANDFFASGPYKGEDLRLTDLAGDWDQADMAG
VPSEYILYFVADWRADLDSLYLDAAVGRASEQISAALEEDGDTSAEDESGELEQAGGTENRE
>lcl|NZ_CP026520.1_prot_WP_042228125.1_2544 [locus_tag=PC41400_RS12970] [protein=hypothetical protein] [protein_id=WP_042228125.1] [location=complement(2813129..2813842)] [gbkey=CDS]
MKKLVSALLAAAVPAVSPATAVYGAAIESEYTDAASGIRFSTFSILSEHRGITDAAVRYTDPKRKQKSPALQSKAALAPA
EQISEVYDHSVYSNGGTSMTSLTSSPRENDIFLVSTARGSSHTLTEPRRISASLQISGVYNSGIVKLVRLAVIGNASGEL
SYVWPPSKVYTAPETVPFHRDYYGAVQYDTSSTQVRQTDIYKQYYKSIYVREVHYDRGIVETVSSKAPKPVEYSIDF
>lcl|NZ_CP026520.1_prot_WP_042228124.1_2545 [locus_tag=PC41400_RS12975] [protein=AAA family ATPase] [protein_id=WP_042228124.1] [location=2814035..2815000] [gbkey=CDS]
MDATSRRITVRSNSQINIVFNSQEAAPLIQEPESTTLDIQSETFNETLKELNTMIGLQEVKDLIYQMYATLHINKLRSAA
GLKDNSQVYHMIFKGNPGTGKTTVARILAKLFKSLGVLSKGHLIEVERADLVGEYIGHTAIKTRDLIKKALGGILFIDEA
YSLARGGEKDFGKEAIDCLVKSLEDHKNDFILILAGYPDEIDYFLSTNPGLPSRFPIQVDFTDYNVDQLIQIAELMMSER
DYTMRLESVNKLRDIIMMEKETSLNFNNARFVRNVIEKSIRNQAVRLLSRYKKPSKQDLMEILPQDVAATENKLKMEYGV
S
>lcl|NZ_CP026520.1_prot_WP_042228123.1_2546 [gene=hflX] [locus_tag=PC41400_RS12980] [protein=GTPase HflX] [protein_id=WP_042228123.1] [location=2815352..2816671] [gbkey=CDS]
MKQTGHEVRTGMKDRAILVSLVTDNMKKHEELVAYSLQELIQLAETAGVEVLETLTQNRERADTKWFIGKGKVEELKALI
DEHEATTAIFDQELSGAQVRNLEEALDVKIIDRTQLILDIFAGRARTREGIVQVELAQLSYLLPRLSGHGKNLSRLGGGI
GTRGPGESKLETDRRHIRGRIGELKAHLQEMVRHRTLHRERRRKTGVYQVALVGYTNSGKSTLLRQLTNADVYVQNELFA
TLDPTSRMLELPSGKEVVLTDTVGFIQNLPHDVVAAFRATLEEACEADFLLHVVDSSSPMREEQMRVVNEVLGDLGASGK
PQAVLFNKIDLCTEQELQLLTTNLPHLRISAYEEKDQERVKLLIQETLMGDSRTFRIPADKGDIIALVYRVGDVAESGAE
EDVLTFRVNINKDDYDKVQYLLSPYEVKEEKDKQDTQGE
>lcl|NZ_CP026520.1_prot_WP_042228122.1_2547 [locus_tag=PC41400_RS12985] [protein=methionine gamma-lyase family protein] [protein_id=WP_042228122.1] [location=2816691..2817941] [gbkey=CDS]
MQFDSDIMAIMEKAEEGVQPRFREIERTIDHNQWKVVAAFQNQGISDYHFASSTGYGYNDRGRELLDLVYAEVFGAEAAL
VRPHFVSGTHTIGTALFGVLRPGDELVYITGKPYDTLHKVIGKPGDGTGSLQDFGIGYREVPLLGDGSLDEAGIRAAITP
KTKVIGIQRSRGYDWRASFTVAAIGEMVKLVKSIRSDLIVFVDNCYGEFTERLEPNEVGADLIAGSLIKNPGGGLAASGG
YIAGRKELVELAAYRLTAPGIGGEVGAMLGTTRSLYQGLFLAPHLVGQAVKGSVVAAAVFEQLGFKTRPRPEDPRTDLIQ
AIEFTSAEHLITFVQGVQRASAVDAHVTPEPWDMPGYEHPVIMAAGTFIQGGSLELSADAPIREPYIAYMQGGLTYSHCK
LGVLTALQHMKERGLL
>lcl|NZ_CP026520.1_prot_WP_009674344.1_2548 [locus_tag=PC41400_RS12990] [protein=MerR family transcriptional regulator] [protein_id=WP_009674344.1] [location=2818135..2818548] [gbkey=CDS]
MSDDIRRNMALFPIGIVMKLTDLTARQIRYYEQHELVIPARTSGNQRLYSFNDVERLLEIKDLIEKGVNIAGIKQVLLPI
SKDSEEATVITEQTEVKRKELTDTQLHKMLKQQLLGGGKRPGQVSLIQGELSRFFPK
>lcl|NZ_CP026520.1_prot_WP_042228120.1_2549 [gene=glnA] [locus_tag=PC41400_RS12995] [protein=type I glutamate--ammonia ligase] [protein_id=WP_042228120.1] [location=2818579..2819913] [gbkey=CDS]
MTVSYTKEDILRIAKEENVRFIRLQFTDLLGTIKNVEIPVSQLQKALDNKMMFDGSSIEGYVRIEESDMYLYPDLNTWVI
FPWVTESKVARLICDVYLPDGTPFPGDPRGILKRALKEAEEMGYTSMNVGPEPEFFLFKTDEKGNPTLELNDQGGYFDLA
PTDLGENCRREIVLKLEEMGFEIEASHHEVAPGQHEIDFKYADAVKAADQIQTFKLVVKTIARQHGLHATFMPKPLFGVN
GSGMHCNQSLFKGGENAFVDEKDPLGLSADARHFMAGVLQHARAFAAITNPTVNSYKRLVPGYEAPCYVAWSASNRSPMI
RIPASRGLSTRVEVRNPDPAANPYLALAVLLRAGLDGIKNKTQLPPPTDRNIYVMTEEERIEQGIPSLPGNLREALDELL
KSEVIGDALGEHALAHFVELKEIEYDMYRTQVHAWEREQYMTLY
>lcl|NZ_CP026520.1_prot_WP_042228118.1_2550 [gene=lexA] [locus_tag=PC41400_RS13000] [protein=transcriptional repressor LexA] [protein_id=WP_042228118.1] [location=complement(2820072..2820683)] [gbkey=CDS]
MGKVSQRQLAILEFIKNEVKDKGYPPSVREIGEAVGLASSSTVHGHLERLEKKGLIRRDPTKPRAIEITDGDSDFALSIA
QVPLIGKVTAGMPITATENIEDYFPLPMHLVSDYNVFMLNVVGESMIEAGIHNGDMVIVRQQQTANNGDIVVAMTDEDEA
TVKRFYKERDHIRLQPENSTMDPLLLNNVTILGKVIGLFRNIH
>lcl|NZ_CP026520.1_prot_WP_042228116.1_2551 [locus_tag=PC41400_RS13005] [protein=LysM domain-containing protein] [protein_id=WP_042228116.1] [location=2820828..2821361] [gbkey=CDS]
MNAYFANPELSLHTRSGDQPATDKASPHTTAANRANDHRENSSRFTSRVSDFKSEQGALKAGERTNVRYAKKSNRNIVRT
AVIAIVLLIAFSCGIMVQAFASGENSPADGQRQAAVLNKQESAEPSVIVQRGDTLWSIAAAQAPEGSDVRAYIYKLRKVN
QLSAGDVLKAGDVIKLP
>lcl|NZ_CP026520.1_prot_WP_042228114.1_2552 [locus_tag=PC41400_RS13010] [protein=DUF896 domain-containing protein] [protein_id=WP_042228114.1] [location=2821584..2821766] [gbkey=CDS]
MSDTFQLKVQRINELARKAKSVGLSEAELEERNELRKEYIAAMKTQLRGQLDSIHYTDEQ
>lcl|NZ_CP026520.1_prot_WP_042228112.1_2553 [locus_tag=PC41400_RS13015] [protein=HAD family hydrolase] [protein_id=WP_042228112.1] [location=2821805..2822617] [gbkey=CDS]
MTVKAILFDLDDTLLWDDRSVKEAFQATCEEAARHADVNPAELEASVRKEARALYEAYETFPFTQHIGINPFEGLWANFV
EGKQEEFRKLQALAPGYRKEAWIRGLAALGVNDPELGARLAEQFPAERRSRPIVYEETFEVLAELKKSYKLLLLTNGSPD
LQREKLAGVPELAPYFDHIVISGDFGDGKPAVTIFNHAMELLGLQAGECVMVGDKLTTDILGSFRCGMKNMWINRHGMLF
NGEAEPTFEITSLRDIRGILEAEREVQISG
>lcl|NZ_CP026520.1_prot_WP_042228111.1_2554 [locus_tag=PC41400_RS13020] [protein=acireductone dioxygenase] [protein_id=WP_042228111.1] [location=complement(2822706..2823248)] [gbkey=CDS]
MAEIHIRNTGERISGEENVRAFLDRQEVLYEHWNPTKLPADIQEKFILSDEEKAQILSTFDEEIRDLAKRRGYQTWDIIA
LSDATPNLDELLKKFEQVHTHTEDEVRAITAGKGIFIIKGSDDVGYFNVELEAGDVISVPEHKPHFFTLMDNRQIVAVRL
FIETEGWIAHPFEDPTFQKA
>lcl|NZ_CP026520.1_prot_WP_042228109.1_2555 [gene=tatA] [locus_tag=PC41400_RS13025] [protein=twin-arginine translocase TatA/TatE family subunit] [protein_id=WP_042228109.1] [location=complement(2823464..2823658)] [gbkey=CDS]
MLNNIGISGLLIILMLALILFGPSKLPQLGRAFGDTLREFRTSTKEMVDDVEKSIETSDRDKQS
>lcl|NZ_CP026520.1_prot_WP_042228108.1_2556 [gene=tatC] [locus_tag=PC41400_RS13030] [protein=twin-arginine translocase subunit TatC] [protein_id=WP_042228108.1] [location=complement(2823703..2824434)] [gbkey=CDS]
MDRPSEMIGHLEELRKRLILVAAWFLLALIGGLYVSPVILNAVKNSPAALSFHWNVFALPDGLAIYMKCAVLFAVLLTLP
LLLYQLWAFVRPGLSEQEAKGTLIYVPASFALFLSGSAFGYTVVLPMMIRFMQTINRSIGAAETYGIDRFFGLVFNVVFP
LGIAFELPLLVLFLTRLGLLTPENLRKTRKYTYIGLAIVGASISPPDFFSHIAVTIPLIALFECSILLSARYVRRMNLSA
GTS
>lcl|NZ_CP026520.1_prot_WP_042228107.1_2557 [locus_tag=PC41400_RS13035] [protein=GMC family oxidoreductase] [protein_id=WP_042228107.1] [location=complement(2824474..2826195)] [gbkey=CDS]
MATKLPKVPILIVGMGWSGGILAAELTKAGHKVVGLERGKSRKTEDYYMAHDELRYSQRYELMQDLSKETVTYRSSEKVR
ALPMRSYGSFLMGDGVGGAGVHWNGQNYRFLPYDFEIRTKTIERYGKNKIPQGMTLQDWGITYDQLEPYFDKYEKMAGIS
GEKEQNEMVGKRSAAFPTGPMKKTPGMVLFEDAAKKLGYHPYAIPSANLSENYTNPDGIARAACQYCAFCERYGCEYGAK
ADPVVTVIPVAEKTGNLEIRTNSNVTKILHTGGKVTGVVYVNTPTGEEFEQPADLVILASYVFNNVRLLLTSNLGKPYDP
KSGTGVIGKNYAYQITGGGAAGFYDDQEFSIFAGAGALGSCFDDFNGDNFDHGSLNFLHGANIRFTQAGLRPIAHNPAPQ
GTPRWGKDFKQTTIKYANRFLNVSTQGANMPWQHHYLDLDPTYKDIYGLPLIRLTYDFEDQDKEMVKYLSEKTTAVMKEM
KPTKVETSNTISTYNIVPYQSTHNTGGVIMGADPSTSAVNSYMQMWDVENLFVVGASAFPHNSGYNPTGTLGALAYRAAE
GILTYLKKSGPLV
>lcl|NZ_CP026520.1_prot_WP_042228105.1_2558 [locus_tag=PC41400_RS13040] [protein=gluconate 2-dehydrogenase subunit 3 family protein] [protein_id=WP_042228105.1] [location=complement(2826215..2827000)] [gbkey=CDS]
MPENQQTKQPRDESRRKFLKLSGVALGGIVAGGVVGGIVGNSMAKNKPAAPGSPQGQGQAQAGNTKDYNQAPMFFNQEQL
RITEAATERIFPKDELGPGAKELGAAYYIDHQLAGQWGMNARDYMMGPFYKAEATQGYQASYKRHELFVMGLTVMNDHSK
QKYQKGFADLAGNEQDEVLKAFESGKDIQLNGVPSQTFFEMLRSLTIEGAYADPLYGGNKNMAGWRMRKYPGDQMSYTAI
IDKKEFTVMEPLSLHDHTNHS
>lcl|NZ_CP026520.1_prot_WP_042228104.1_2559 [locus_tag=PC41400_RS13045] [protein=cytochrome c] [protein_id=WP_042228104.1] [location=complement(2827186..2827608)] [gbkey=CDS]
MNKRMTALVGILLVTASLAACGQKQTAPTPQPTTPASTAPPAASPTASPGAAAGGNAPEGAGSGSQTAQAESLFKAQCIA
CHGADLGGGVGPNLQKAGSRLSEADVKNIVLNGRGGMPAFKGNLSDTDIQALAAWLAAKK
>lcl|NZ_CP026520.1_prot_WP_042228102.1_2560 [gene=metH] [locus_tag=PC41400_RS13050] [protein=methionine synthase] [protein_id=WP_042228102.1] [location=2827860..2831303] [gbkey=CDS]
MKPSLKEQLQKKILILDGAMGTMIQQEDLSAEDFGGEELDGCNEMLVLTRPDVIQRIHEKYFEAGSDIVETNTFGATSVV
LAEYDIPEKAREINLAAAKLAVEAARKYSTPEWPRYVAGALGPTTKTLSVTGGVTFEELEESYYEQTIALIESGVDAILL
ETSQDTLNVKAGGIAIQRAFKTTGIELPIMISGTIEPMGTTLAGQNIESFYISLEHLNPISVGLNCATGPEFMRDHIRTL
SGLAGTYISCYPNAGLPDENGHYHESPESLAKKMAAFAEQGWLNIAGGCCGTTPAHIRALAETLSAYKPRTELGTHPSAV
SGIETVYIEPENRPYMVGERTNVLGSRKFKRLIAEGKYEEASEIARAQVKGGAHVIDVCLQDPDRNETEDMTKFLELVVK
KIKVPLMIDSTDTRVIELGLKHSQGKAIINSINLEDGEEKFEKVLPLIHRYGAAAVVGTIDERGQAITREDKLEVAQRSH
DLLVHKYGMNAEDIIFDPLVFPVGTGDQQYIGSAKETIEGIRLIKEAMPATKTILGLSNVSFGLPEAGREVLNAVYLYHC
TKAGLDYAIVNTEKLERYASIPEEERTLAEKLIYETNDETLAEFVAHFRVKKVEKKEKISTLTLEERLASYVVEGTKEGL
LPDLEQALQKYSPLEVINGPLMKGMAEVGRLFNNNELIVAEVLQSAEVMKASVAYLEQFMEKNETAVKGKIILATVKGDV
HDIGKNLVEIILSNNGYQIVNLGIKVPPEQLIEAFRKEKPDAIGLSGLLVKSAQQMIVTAQDLRNAGIDVPIMVGGAALT
RKFTKTRISPEYDGLVLYAKDAMDGLDIANKLSDPVQRERLIQELRESKDSDVMEAGRKEGTEMPVLTRVFSSTVDRTLP
VSVPPDLDRHVLRDYPIGHLMPYVNMQMLLGHHLGLKGKVDKLIAEQDPKAMQLKDTVDTILYEAQKNGIIRSQGMYRFF
PAQGDGNDVVVYDPEDTSKVIKRFTFPRQGTEPFLCLADYLKPVDSGVMDYIGFLVVTAGQGVSELAREWREQGDYLRSH
ALQSTALEVAEAFAERVHHIMRDVWGIPDSAEMTMQERFGAKYTGQRFSFGYPACPNLEDQEPLFELMKPEDIGIELTEG
CMMEPEASVSAIVFAHPQARYFNVDKV
>lcl|NZ_CP026520.1_prot_WP_042228101.1_2561 [gene=rnz] [locus_tag=PC41400_RS13055] [protein=ribonuclease Z] [protein_id=WP_042228101.1] [location=2831559..2832494] [gbkey=CDS]
MELYFLGTGAGMPARHRNVTSIVLNLLPERGSIWFFDCGEGTQHQILRSPVKLSKSEKLFVTHLHGDHIFGLPGLLSSRS
YQGGDTPFTIYGPKGIKAYVDMSLQLSQVHLDYELAIREIPSEGGVVFEDESFRVEAAPLDHRIECFGYRIVEKDLPGKL
QQEKLHELGITAGPLYGRLKQGQTVKLEDGRELHGSDFVGPSIKGRTVTILGDTKPCGNSALLAEGADVLVHEATFGAER
EDLAGAYDHSTTEQAARTAAEGGVGTLIMTHISSRYQGEGAERLLEEARAIHPDSWLAEDFYSHAVPRRQS
>lcl|NZ_CP026520.1_prot_WP_042228099.1_2562 [locus_tag=PC41400_RS13060] [protein=spore protein] [protein_id=WP_042228099.1] [location=complement(2832724..2832951)] [gbkey=CDS]
MAKKDNRGDNAVHLRNSIRNTEARLQESQEYLDEHADEISATEQATLEAKNERRRQSVRGFKSELQDEAKFETTE
>lcl|NZ_CP026520.1_prot_WP_042228097.1_2563 [locus_tag=PC41400_RS13065] [protein=metal-dependent hydrolase] [protein_id=WP_042228097.1] [location=2833113..2834087] [gbkey=CDS]
MDTGSHLLFGITLAGLATLQPEVAHDPLLLQAVLAATVAGSHAPDLDTIARLKGYAVYLRHHRGITHSLPALLVWPALLS
ALAAAVFGTWDHYALLYGWAAIAVILHVGLDLLNAYGVQCLRPFSSRWLHLDILPLTDPFLLALHGGGALLWVAGAANPE
MWMAAVYGASFTYVMARWLHKRILLRRLTGYAKDLLGAAGPLQLVPGLTWRSWQFAAETEHSFHSGTVRGSRIVLKDTYV
KDRTHPVVEAAMAAEGVRAFLDFAQSSHVSWDSHGEGYVVKWREVRFWHEHKLPFGIDVTLDREMKVLSEQLGWTKKAWE
PPYV
>lcl|NZ_CP026520.1_prot_WP_042228095.1_2564 [locus_tag=PC41400_RS13070] [protein=nitroreductase family protein] [protein_id=WP_042228095.1] [location=complement(2834132..2834734)] [gbkey=CDS]
MPEHRFDVQNERHAAYDINPIFLNRWSSRAFLEKEISNEILMSLFEAARWAPSASNLQPWRYIVARTEEEKGMFAEFISA
SNKTWCLKAPVLALILSHTPSPAGGVNGSHAFDAGASWSYLALEAVRQGLISHAMAGFDRESARRLLGVPDDYDIHAVIA
IGYHGDAQSLSDQHLQREQPSGRRELKETLFAGRFGNSFE
>lcl|NZ_CP026520.1_prot_WP_042228093.1_2565 [locus_tag=PC41400_RS13075] [protein=hypothetical protein] [protein_id=WP_042228093.1] [location=2834849..2835106] [gbkey=CDS]
MLVQLLFNKMGVLNMNFTSQDIIVIERALQLAIRSETKETKAEEYREVLHKLQLSSMQALQEQSAVTTAVQDGIRYDYDD
SSDLV
>lcl|NZ_CP026520.1_prot_WP_241688857.1_2566 [locus_tag=PC41400_RS13080] [protein=hypothetical protein] [protein_id=WP_241688857.1] [location=complement(2835079..2835324)] [gbkey=CDS]
MNGRFHVWSTIGPMSRSHGLKPAFFRPSIADRRIQGLYDAGGTTHPGKGTPTAAPSGMPAAEAVSRSYPFRDAYTRSDES
S
>lcl|NZ_CP026520.1_prot_WP_042228088.1_2567 [locus_tag=PC41400_RS13085] [protein=DEAD/DEAH box helicase] [protein_id=WP_042228088.1] [location=2835832..2838078] [gbkey=CDS]
MRKKQLAELIQELRTDENIVNWHEIEPQEAKTKAIPGSVDARIKSALAKRGITELYTHQHTAYETVHTGQNIVAVTPTAS
GKTLCYNLPVLQAIADDDTSRALYLFPTKALAQDQKSELNEIIGEMGIDIKSYTYDGDTSPAIRQIVRKAGHIVITNPDM
LHSAILPHHTKWVSLFENLKYIVIDELHTYRGVFGSHVANVIRRLKRICKFYGSDPVFICTSATIANPKELAEQLTGSPM
RLIDDNGAPRGRKHFVFYNPPVVNKPLNIRRSAAAEVNNLAKQFLKNKIQTIVFARSRVRVEIILSRIQELVKNELGTKS
IRGYRGGYLPKQRREIERGLRAGEILGVVSTNALELGVDIGQLQVCIMTGYPGSIASTWQQAGRAGRRHGEALIVMVASS
TPIDQYVVQNPDYFFDRTPEAARINPENLIILVDHLKCAAYELPFKETEEFGPLEVAEIMEFLVEERVLHHNGDTFYWAS
QSFPANEISLRSASQENVVIVDQSDVANVRIIGEMDRFSAMTLLHDEAIYLHEGVQYQVEKLDWEHKKAYVREVDVEYYT
DANLAIQLKVLEIDKTSDRDRSSLNFGDVTINAIPTIFKKIKLTTGENIGSGPIHLPEEELHTSAAWVELKEPNLEVGTK
ALEQILWGIANVMKHIVPVLVMCDRNDVHVISQIKANHTGLPTIFLYDHYPGGIGLAEEVYTRFDEVRKAAKNLIQKCPC
EDGCPSCIGSEIEGIHAKRTSLQLLDQF
>lcl|NZ_CP026520.1_prot_WP_042228086.1_2568 [locus_tag=PC41400_RS13090] [protein=GNAT family N-acetyltransferase] [protein_id=WP_042228086.1] [location=2838130..2838603] [gbkey=CDS]
MVALEPMLPEQFNEWLQLSVKEYAESKTKSGNVAAEKAREDSAREFKELLPDGLNTADSYIFNATDEESNHIVGTLWIKI
RSEQREVFIYGIRMSEDFRGKGFGKQTLQALEAFIKERGFPLIISLHVFGDNEIALNLYRSVGFEATNIRMSKVISG
>lcl|NZ_CP026520.1_prot_WP_042228085.1_2569 [locus_tag=PC41400_RS13095] [protein=ribonuclease H-like domain-containing protein] [protein_id=WP_042228085.1] [location=2838816..2840534] [gbkey=CDS]
MSGLRDRLARHLKPGKTAAAPEAGAEQRQAELTPEQAQHGPAEGGGPAVCPLSESAAGAVAGPAREPAAAPVAAGNAPAE
GSAPAEGAHGYAGHASGGSPADARSGSGQAVEPAASAAPAAPAPAHPEAAAWAKLDAELYEFPGGSFVRRVRTYPMDAYH
GRYRLDALHEHVREFGAFHPEAEPVHAEGLLFFDTETTGLGVGAGNVAFLVGIGYYETGRFVVEQLFIRHPAEERAMLAY
LEEKMNRFTHVVSYNGRTFDWPILQNRFILNRMRPDIDRLLHLDLLHPSRSLWKHSMPSVRLGKVEEAQLGYAREDDVSG
AFAPELYVLYLAEGKTDVLRGVFVHNERDIVSLAALSALITRYLADEEPLEPKSPGDLFRLGQWLLKMNRLPSAEKILEL
GFSRLQEQGEEGRTVSGDLLLRYAAYYKKAGVRDRAVELWNRLIRQAGPAAAAPVEAYIELAMHYEHGCKDYALALELSR
EALDVLRQRRRLLRRSEKAADEERRIAHRIERLQRKLRDCGKSVSEKKASVKAAIPSRPPKPPGQARSAKPSGKPRPAYA
ADSLFDVARNPG
>lcl|NZ_CP026520.1_prot_WP_042228081.1_2570 [locus_tag=PC41400_RS13100] [protein=DNA-formamidopyrimidine glycosylase family protein] [protein_id=WP_042228081.1] [location=2841015..2841824] [gbkey=CDS]
MPEWPEMEHYRQELNKRIQNRTISGITVNRPKSLNVPEEEFRAAVLGAVVERVDRRAKQLVFVLSGEHYMLLHLMLGGMM
VLGTEADKPERTTQIELHFGETVLYFIGLRLGYLHLYGKRGLEAELASLGPEPFDADLTAGRFADMLKTGKGNLKTSLTD
QHWLAGVGNCYSDEICFAAGLLPSRRPAALEEHEAARLYTSMRSVLTEALRAGGYMEMPLYDGDERTGGFNELCRVYDRG
GEPCVRCGNPIVFRTLSSRKCFFCGSCQT
>lcl|NZ_CP026520.1_prot_WP_042228080.1_2571 [locus_tag=PC41400_RS13105] [protein=deoxyribonuclease IV] [protein_id=WP_042228080.1] [location=2841849..2842721] [gbkey=CDS]
MVREKKKDLHIGAHISTRKGFYGAAVTASAIGAGAFQFFPKNPRSLILKTNVDWSDARRCAEYSRENGLLSIGHAPYPLN
PAAEGGQQDVMVEAMLNALEIVDACGSVGLVVHFGKFSGKDPLQGYKNIIQCTNKTLSRWRGSALLLLENQAGSGSQMGT
TMEELARIRELSEYPDKIGFCLDTCHAFAAGLWHGDDWDEVEKRGGELGYFDHLKAVHLNDSLYPHGSGKDRHANIGDGY
IGLEGFKSLLASDVVRGIPLVLETGSGPDGTHRKEIALVKQTAGAAGKES
>lcl|NZ_CP026520.1_prot_WP_042228078.1_2572 [locus_tag=PC41400_RS13110] [protein=hypothetical protein] [protein_id=WP_042228078.1] [location=2842721..2843053] [gbkey=CDS]
MKIHVFLDDQRRCPAGFTLARNVEECRLLLQDCEVDILSLDYDLGWDESGWDIVAWMIGTGTFPARIYLHTSSLEGRNRM
YQALYASVPSTTSLHPGPPPAGLLAELAGG
>lcl|NZ_CP026520.1_prot_WP_042228077.1_2573 [locus_tag=PC41400_RS13115] [protein=ATP-binding cassette domain-containing protein] [protein_id=WP_042228077.1] [location=2843094..2843858] [gbkey=CDS]
MIHLNDIHFIREERHILQDVSFHAEKGEHWVLLGKNGSGKTTLLELMNGYQFPSRGKVEVLGNRYGQVDVREVRKRIGYI
SQSLFEKMNLADTVWEAVATGEYAYLRFYQEIPEDVKKKAIGMLERIGLPHVAMQSIGTLSQGERKKVMLARSMMQKPEL
LIMDEPCSGLDLYEREKLLAAIREFSDQDMTVLYVTHHIEEIIPMFTHVALIEQGRIIAAGPKEEVLTPEHLYQAFQVPL
EVEWFRNRPWIKVL
>lcl|NZ_CP026520.1_prot_WP_042228074.1_2574 [locus_tag=PC41400_RS13120] [protein=SAM-dependent methyltransferase] [protein_id=WP_042228074.1] [location=2843862..2844908] [gbkey=CDS]
MTEHDSTTNTARLIGTANRGFAQHAQEEVRRLIPGVKFRTLVPTETFLAEAPGSWEEISEAIRAQEPIFLRHMQPVQGGR
VLSGTDEDILAAAELASELGGLHAGSSSAVQVRIADGTEVPYGASDVRAACEIAMSEATGAETLVRHADKILSVYVGHQA
IYAGVSAPEENLSDWSGGAIRFQREENQVSRAKFKLLEAERKFHLHFADFRKAVDVGAAPGGWTSLLLERGLHVTAIDPA
KMHPSLIKHPRLTYHGKNAADVKLREGEFDLLVCDMSWSPRQMSRLILDLLPALQSGGTAVITAKLMHKKPFQTVKELKQ
DFEGHLELRQAKQLFHNREELTLYFVKK
>lcl|NZ_CP026520.1_prot_WP_042228073.1_2575 [locus_tag=PC41400_RS13125] [protein=tryptophan-rich sensory protein] [protein_id=WP_042228073.1] [location=complement(2844920..2845390)] [gbkey=CDS]
MFYPFMLAAVVTYGLFLLSAALFPADMRWYSSVKKPAWAPSGQLIISARGFMHACMAASVALVYVKTGGFQNVGIAWIFV
VILHYLAGQAYGFFLFKAKSFGLALADSMIAALTGWWLLVLTAPLSKEAAWLLLPYLVWSCLTTLFTWIIRIMNTE
>lcl|NZ_CP026520.1_prot_WP_042228072.1_2576 [locus_tag=PC41400_RS13130] [protein=hypothetical protein] [protein_id=WP_042228072.1] [location=2845735..2846862] [gbkey=CDS]
MKRLLILIGLDKEFASAMERFIREGEYCERLNIRSFTHCEHAGTFVNDNRGESPVILHGETIADEEVIRWFGEGHNGLVY
RLSETPECTDLPAGNRIYRYQPFRTMLRDLLLRAAEQTGVPGPGRAGFRSGRTSILTVYSAFGGSGKSVLALNMATQLAA
QGRKVLYVSLEAVSSSALLMKDGPPDSLSRLLYAWRKSPDRTADYADLLQCRHPAYGFDYLAPLPNIREADELQAEDVSG
LVGRIADSGICDVLLLDLDSAVQPRTLGALRASDIVLWLVTQDAHALYKTRKVLEAMKSMLSMESCANKSLFVVNKYTGG
DLGLAKEYDVAVFETLPYIPAWKTVSDPADWLGEPVFAAQVGHMLRRFSEGAEAL
>lcl|NZ_CP026520.1_prot_WP_042228071.1_2577 [locus_tag=PC41400_RS13135] [protein=CpaF family protein] [protein_id=WP_042228071.1] [location=2846859..2848115] [gbkey=CDS]
MNEEWVRTFKEAVNARLDPTQSTSDGELMELIEEVVLRETGDGRLTAGEKWEMVQRIFHSYRGLDILQPLIDDASVTEIM
INSHRDIFIERDGRVSRNEARFESREKLEDIIQAIVAKVNRTVNEMSPIVDARLQDGSRVNVVLPPVALQGPTMTIRKFP
AQPLTMSDLIAVGALTRQAAELLGVFVKAKYNIFIGGGTGSGKTTFLNALSQGIPVDERIITIEDSAELQIRTVPNLVSL
ETRNANTEGKGGIGIRDLIRSSLRMRPNRIIVGEVRGAEALDMLAAMNTGHDGSFSTGHSNSTADMLSRLEMMVLSAADL
PVEVVRKQISSAIDLMIHLSRMRDGSRKVVEISELAGIANGEIFLNPLFRFEDQGEDENGKVKGELFYTGNSLVHGWKCR
MAGVELPGWITREKEAAS
>lcl|NZ_CP026520.1_prot_WP_042228070.1_2578 [locus_tag=PC41400_RS13140] [protein=type II secretion system F family protein] [protein_id=WP_042228070.1] [location=2848112..2849044] [gbkey=CDS]
MMLPVAACLAGLVLWLVLLTQDKRKRRKWAKGGDNGGEKGYARQLGNEIRLPERKTEARPAPAGLTDYGSYRMTGREKLL
TSGLSGLGFAGIAFIFYKEPLLAILCALLGLLVPRFRKVRLITGRKAKLILQFKQALYCMSTSLTAGRSVENSIREAWKD
LQLLYPDPNCLIIREFEIIVRRLDNGEPIEHAVADFAVRAQCDDITNFSDVFTTCKRTGGNLVEVIRRTAGIIGDKLDIE
QDISVMVAQKKLESRVLLAAPVCMVAVLSFSSPDYMAPLYSGAGPLIMTGGLLALGGCLWLTGKIMDIKV
>lcl|NZ_CP026520.1_prot_WP_042228069.1_2579 [locus_tag=PC41400_RS13145] [protein=type II secretion system F family protein] [protein_id=WP_042228069.1] [location=2849049..2849918] [gbkey=CDS]
MLVLLCLSMGSFLALLLLYALGSRRFKDVRTDYPCTLHMKKLAPAGLYAVERFKTMERFPDAGVRIMQRIRSLYGHKDTL
VMAKLVMGDMLLTALIVLVLTGLLGIAGGGDPAVALLGVLGAGLTPWLMLRQLDQQIRLRKRRIVLELPEVLSKMILLIN
AGETVQKALIRSVDTSRVGASPLHAEFGQAVLELEMNVSLPKVLEDLSKRCAMQEMALFTTTILLNYRRGGEDFLVALRS
LAKELWDRRKAESRKLGEEASSKLVFPMVAIFLVVMIIVASPAVLSING
>lcl|NZ_CP026520.1_prot_WP_042228068.1_2580 [locus_tag=PC41400_RS13150] [protein=Flp1 family type IVb pilin] [protein_id=WP_042228068.1] [location=2849958..2850167] [gbkey=CDS]
MKNRLQTSMIRFWNDEEGLGTLEILLIVAVLIAVAVIFRKWIIKWFNELIGKADTEFNNSKSDDAIKTP
>lcl|NZ_CP026520.1_prot_WP_241688858.1_2581 [locus_tag=PC41400_RS13155] [protein=hypothetical protein] [protein_id=WP_241688858.1] [location=2850241..2851173] [gbkey=CDS]
MEGTLLFPVVILITLTLIIFGMFVYNRVALQQQAGVASERAAFSWNNSRKDPATGAFTPGAGDGLYWRLTQDGILGIFGY
SGKSSVVHVPDTSVGSGPAGKLSRAAALLPPGVHGSMSYVNHIWERKVSAELEKPFRTRSFLPGNVIRDQVTGQGASHVV
DPVELIRTVDLTRSYIPAIKNRITASKAREVLTEHAAEAIPPTNAITSEAQASAYIRKLVSGRKTELITPEGDKRTIDAI
DAGGTAHLAFYTFNEKNLKEQMRKDILLKKEGTQVKGIVWHFFKTKSGKAPSQAIQRELAQNGISVILHE
>lcl|NZ_CP026520.1_prot_WP_042228067.1_2582 [locus_tag=PC41400_RS13160] [protein=hypothetical protein] [protein_id=WP_042228067.1] [location=2851212..2853470] [gbkey=CDS]
MRSFYNSETGAVSIYLIIILVPIFFLQAVLIDFARVKAAEREADTAARAAVRSVLSAFDTELQKYGLFGLTGEPESSESL
FSMIYKENLSGSSSGGFYKLLDTKPEEGHASLTSLYTLANHTVFKEQILEHMKYTAPIEFVLEITDKLQKPGAVKTLANG
SAYAKDAEKIEKLLVKREELLDRAWMETVKLKEELNGRHHTYRARITELNRLSELIGLNTADGVRSQIKTLEEQIRSLSD
SISSLNGSLAAAAAAGPEAAAAIQGMMDNMRSLQDQLTELQRRLSDLKTLLDNIVKYTGLLAATKLEIVSDDEKVQNLQK
SVQELLGEAKKANDELNTELKRVSSAAGNELKVFEAFKSLKILSEDDFRTYQTGIASSAAMFSGFRSGIGSINLFTNANA
QKIHAANDAYQAQMDEIYQKQLVIETERNKRNETIRKEKEKQRSKIQAALDRARQAMGDCGTDQTAGSVSAFYETLEGKD
GLYGKYMTLNESPSNAGGSPVYDLEGKPDTEIQHAMSLLERLGQTGESIRNELFVNEYALSLFNYRTYGREVDGSGQPKP
LHAMSEPSGHKLPGQEVEYLLYGFNSCAKNRSSAYGEMFAFRFAVRMVEELMDPKNELLQAGSPLLVLLVSAAEAAVKAL
ADMDSLTDGKAVPISSKLTNPLFTFTYKDYLRIFLLLHSSDTNLLARLQSLIELNTERTLDHAVTYIQGSAESSVSLWFL
PGVMKLMNGTGISSCEVRENRCRFISRADLAY
>lcl|NZ_CP026520.1_prot_WP_042228065.1_2583 [locus_tag=PC41400_RS13165] [protein=hypothetical protein] [protein_id=WP_042228065.1] [location=2853511..2854149] [gbkey=CDS]
MRMFRKSEQGGMVLEAALILPMFVGFIVALIVCIQIAVADMALRSVVSESTKVLAAGLYPAEVLYKQGKESFDQTALASG
GQQVMDTLQNLREQAAQAGEFIDSVEAYLPDPVIGLLSWVRQTGESGVDGAKESASSALRDAMKPIVYSFADKRILKKER
FRISGVTLPSLTSRERAYLGFEVTYDFPLAVPFFNRTLVLKKKALERVWLGA
>lcl|NZ_CP026520.1_prot_WP_042228064.1_2584 [locus_tag=PC41400_RS13170] [protein=A24 family peptidase] [protein_id=WP_042228064.1] [location=2854164..2854679] [gbkey=CDS]
MEELLLSLLILAAFITDVRKSIIPNVLTAAGAGIGMMYHLLIDGWSGLVFSVAGFMTGFIALFILYAAGALGAGDVKLFG
AIGALTGGVYVFTCALYSLLFAGAIGLCLLLARKEGVKRLGRLFGGLALGLFLKDLSFWKSLKQSDQLRFPFMYAVLPGV
LAAYWYGSLAF
>lcl|NZ_CP026520.1_prot_WP_042228063.1_2585 [locus_tag=PC41400_RS13175] [protein=DUF6382 domain-containing protein] [protein_id=WP_042228063.1] [location=2854714..2856459] [gbkey=CDS]
MEIFGLQASVRLSQGRWLEVGLEEGLNIEHLAPLQLKMLKENPIPRVLSLKVEEIDSHVRLIYNLASRRSLAAILKERSP
DPAGCLQLLHEAAACVSDSGNFMLNEAQFVVDKNYIYIERGWADVSLLCLPIKEIPAASASVMWSALVRDLGRRLQAEER
GSLERLAKYCERESFSVREFKRQIYLEISENRLSIEEDIGKGGMESDPIVQASNIPNSEMQGSAEFSERSGYAVSVLKKL
GQDIHETDPLHFQRESNDVGDGLLRRESAHVPQGGTRSPHRLVGESRSDPNPRGAAEKFIRRRQQIAADDSGVQAAGQSP
QLHDTPHAVPQPYAAPRPPQPPQAAPAGAATSAGPLRGSRGKILALVGAFPIIAMIWNAYFSAPGPPLLYISSGATLLVL
DAAYVLLAIGLPGRRNSQLKEAAPSGSSPYPPAAAQEPESLERYYAGLGARTTLLAPPLSAVNNADETVFLGAASAPPHA
KFRHPVLEVMREGRKETILLDADSFIIGRGGGDADYIEDVLGISRIHAQFLRTPEGYAVKDLGSRNGTFLNDGMLVPYQM
VALKDGDVVRIIHTEFSFRFG
>lcl|NZ_CP026520.1_prot_WP_042228061.1_2586 [locus_tag=PC41400_RS13180] [protein=Dabb family protein] [protein_id=WP_042228061.1] [location=2856533..2856823] [gbkey=CDS]
MITHVVFFKLKDRSPESVEQARAKLQNMEGKISELLHIEVGADVIHSERSYDLVLITKFNSLEDLKAYNVHPVHQDVLAY
MGEAKDGVAPCVDFES
>lcl|NZ_CP026520.1_prot_WP_042228059.1_2587 [locus_tag=PC41400_RS13185] [protein=CAP domain-containing protein] [protein_id=WP_042228059.1] [location=complement(2857337..2858155)] [gbkey=CDS]
MKSIKKATLVGVFALTSVFGAGAASASAEPCNVSGGNTLATLSNKYGIDLASLLTSNPQLNNTNFIWKGMQIVVPAAPPA
AEKPAAEAPAAPTAPEKKPEAQKPSAPPVKPSTPVPETQAPVSTPKPTTKPSAPVATPAPKPTEKPASSQSEFADQVVTL
VNQERSKAGLKPLASDSALAAMALDKAKDMYTNNYFDHNSPTYGSPFDMMKSYGISYTYAGENIAKGQRTPQEVMNSWMN
SQGHRENILSPNFTKIGVAYYNGEWVQEFIAN
>lcl|NZ_CP026520.1_prot_WP_042228057.1_2588 [locus_tag=PC41400_RS13190] [protein=M42 family metallopeptidase] [protein_id=WP_042228057.1] [location=2858841..2859929] [gbkey=CDS]
MEQVNRTYLLEVLDMLLQTPSPSGYTHHIMDKVKAEAEKLGYKLELTNKGCGIITIPGKRSDYTIGLSAHVDTLGAMVRS
IKDNGRLRLTSVGGFMMAAIENEYCRVHTRQGLSYEGTILTTKPSVHVFPDARDHKRVEENMEVRLDELAFNKEEVQKLG
IRVGDFVSFEPRVRMLENGYVKSRHLDDKASVAALFALLDLLKNELIEPACTVKIILTTYEEVGHGASYIPADIDELLAV
DMGAMGDDLTCTEQDVSICAKDSSGPYDYAMTSQLIRLAEHSGIRHVVDVYPQYGSDASAALRGGSNIRAALIGPGVHAS
HGMERTHVDAIAGTASLLLQYILHPAAAASDPGQPENPVSPV
>lcl|NZ_CP026520.1_prot_WP_042228055.1_2589 [locus_tag=PC41400_RS13195] [protein=ABC-F family ATP-binding cassette domain-containing protein] [protein_id=WP_042228055.1] [location=2860038..2861987] [gbkey=CDS]
MSIIHAEHLVKTYGEKVLFDDISFTISEKQRIGLIGVNGTGKSSLLRILAGLETPDSGSIVHSNHFRVEYLPQQPEFDER
STVLEQVFYGETPLMKVLRDYEQALAALENSPEDEAAQQKLMRVQARMDELDAWEAGTTAKTVLTKLGITEFGKPVGELS
GGQRKRVAMARVLIQPADLLILDEPTNHIDNETVDWLEQFLSKWKGALLLVTHDRYFLERVTSRILELDRGSIYSYEGNY
TTFLEKKAERMEREAASEDKRRNLLRRELAWLHRGAKARTTKQKARVERAEELRDRKVDGPAAKLDMALAGSRLGRKVIE
LKQVSKGYEGRELIHDFSYIVLPDDRIGIIGRNGSGKSTLLNMLAGRLSPDSGEIERGSTVKLAYYTQENVEMDDKQRVI
EYVREAAEVIRTKDGESITAAQMLERFLFPPNLQWTPIGKLSGGEKRRLYLLRTLMGEPNVLLLDEPTNDLDIQTLSILE
DYLEQFPGAVITVSHDRYFLDRTADHLFSFTGGGEIEQYIGTYSEYLELKQQREASELKPDAKEKARTQASAPSAPSTGT
AGAVSGGERAKKKKLSFNEQKEWDTIEDRIAALEEKLERIKSDIAKAGSDYSRVQELYAEEIAAGEELEQAMERWTILSE
LVEEIERGK
>lcl|NZ_CP026520.1_prot_WP_042228850.1_2590 [locus_tag=PC41400_RS13200] [protein=ATP-binding cassette domain-containing protein] [protein_id=WP_042228850.1] [location=2862093..2863709] [gbkey=CDS]
MITVTDVTLRYGKRALFEDVNIKFTPGNCYGLIGANGAGKSTFLKILSGEIEPNKGDVHIAPGERMAVLKQNHYEFDETD
VLKTVMMGHARLFQIMEEKDALYAKSDFSEEDGIRAGELEGEFAELDGWQAESDAAELLIGLGIPKELHDQKMKDLDGNE
KVRVLLAQALFGSPNILLLDEPTNHLNLESIQWLENFLARFEGTVIVVSHDRHFLNQVCTHIADIDFSKIQMYVGNYDFW
YESSQLALKLAREQNKKTEEKRKELETFIARFSANASKSKQATSRKKQLEKLTLDDIKPSNRKYPFINFKPEREAGKQLL
AIDGVSKSVEGQKILDNISITVNKGDKIAFVGPDGHAKTTLFQILMGETEADEGTYTWGVTTTQAYFPKDNSQYFDGVDM
NLVDWLRQYSKDQDESFIRGFLGRMLFSGEEALKKASVLSGGEKVRCMLSKMMLASANVLILDEPTNHLDLESITALNNG
LIDFDGTMLFVSHDHQFIQTVANRIIEITPKGVIDRMMPYDEYLENEDVRRLREEYYA
>lcl|NZ_CP026520.1_prot_WP_042228054.1_2591 [locus_tag=PC41400_RS13205] [protein=hypothetical protein] [protein_id=WP_042228054.1] [location=2863872..2864405] [gbkey=CDS]
MFDPTVYENLKVVLEGAVYDLDLSGEILVTNRSDVMDLSSLGRQYRIRFVLRGGEENTASPWAELHLAAELEDLAAELLS
RKGAEPGCSLTVRFGFLVGDPSRDCPAVGSSLESIWSGRPLITQTLSYVFGSENEGWHNEVLLEFGRKVNESHLDDFEGL
LEHTLLSLSALRDNHAF
>lcl|NZ_CP026520.1_prot_WP_042228053.1_2592 [locus_tag=PC41400_RS13210] [protein=thioredoxin family protein] [protein_id=WP_042228053.1] [location=2864525..2864887] [gbkey=CDS]
MQEWNDDQIKQALIEKQSFAVFFHTPFCGTCKVARRMLDVALTIEPELPLKACNANYAGASVQKWKISSIPCIVVVEKGT
AARFIYAVKSVDYLLAELRHLTGMQHLRVVASSDFPDNTP
>lcl|NZ_CP026520.1_prot_WP_042228051.1_2593 [gene=kapB] [locus_tag=PC41400_RS13215] [protein=sporulation phosphorelay system protein KapB] [protein_id=WP_042228051.1] [location=2864907..2865305] [gbkey=CDS]
MNYRNESTASVGDLVIASYKTGEYIGRIVEPPSNIKAAVQVLAVVKHPSQGDLHNPQQGDVPFFHQRRALAYQEIGLMPL
SGIHAYHGNVPDYTLSLQQALERELNMLHQTEEWARRSIAELEELHREYFPQ
>lcl|NZ_CP026520.1_prot_WP_042228049.1_2594 [locus_tag=PC41400_RS13220] [protein=hypothetical protein] [protein_id=WP_042228049.1] [location=complement(2865578..2867107)] [gbkey=CDS]
MTSWKKHGPDPRRSKGRLVQTLTLTILMTFLVHLLSAVPALAQEDLALSPPPHSKAAEKKRKEAALKGPSIQVDPSFPYY
QNRSPESIADELLQRGYKTVHYFVVNETDVNRSFIEALQAKGIGVWALVLGNGSYSTARFPADWPSWQMELLKPLNDGYY
RFSPHSPAYVQWKKQALAKLVRDYPFDGIEVAEPYFPEWDGINRGVYGDVGPLAQKAFRDKTGLEMPDFTNASSPRYYLT
DTAAYLKWVDFRVEAVNDFLNELMNGKGGVREARPNILVATWSLAIDAGPGSFDKLRELQGLDAAAMVGRVKPDIHFLQT
HWPDWLKPQESLPPDYVKHYRVFADKIKAEYPDLPLGVQADIGSALSMVKDGGWVSAFSDSVRALGYSTWTAYEYHLGGY
MYKEAPVPGKAQRLSRTEVKIPFQKRIDEGSAKSEGNVTVSAGTQSFTLNPGEISVDGSNLLIRSDRLPKKEFRLTLRGI
KDTPGFWFYNKTQTPNVIGGNVAVEVPKQ
>lcl|NZ_CP026520.1_prot_WP_042228048.1_2595 [locus_tag=PC41400_RS13225] [protein=Gfo/Idh/MocA family oxidoreductase] [protein_id=WP_042228048.1] [location=2867377..2868459] [gbkey=CDS]
MALNDEIRIGVIGAGNIGNVHILGLLETPGVTVAAVADVFLPLAEARAKEHQISKVYGNGEDLIHDPDIDAVVIAVSNEW
HAPLAVKALQAGKHVLLEKPMAIDVKAAKEIVLAERESGKVLMISHQRRWEPLNMQVKELVQRGKLGNIYHAKTGWLRRK
GIPGWGTWFTQMSKSGGGPLIDIGVHMLDLSLSLMGNPRPVSVFGATYSEFGPKRRGIGSWGKPNWSGSYDVEDLATALI
KMDNGSTLTLDVSWAALTETDNQPYIHLLGTEGGATLRGSTYGKLFAEMFDKEIDVELQLPEQQTDPRLLMNMHFLECIR
TGSEPLTSVMSGYTNNLILNAIYESSRTGKEVELDWDLSN
>lcl|NZ_CP026520.1_prot_WP_016424035.1_2596 [locus_tag=PC41400_RS13230] [protein=trimeric intracellular cation channel family protein] [protein_id=WP_016424035.1] [location=2868480..2869160] [gbkey=CDS]
MDWMALSLIGTIAFAVSGAIVAMEEDYDILGVYVLGFVTAFGGGVVRNLLIGEKVTMLWQQGFQFTIALAAMTLVFIFPL
AWIKRWKSWESFFDAIGLAAFSIQGALYASNQGLPVSASIVAAVLTGIGGGVIRDLLAGRKPLVLRDEIYAVWAMLAGFL
IGIGWPQNPLGLIILFMIVVVSRMFSVIFKWKLPRRSLRTLLQQNDVPNKQRPPALQEERREFDVS
>lcl|NZ_CP026520.1_prot_WP_042228047.1_2597 [locus_tag=PC41400_RS13235] [protein=hypothetical protein] [protein_id=WP_042228047.1] [location=2869150..2869620] [gbkey=CDS]
MSAESFSAHGKISGDLLQERLETVKDLKQSELEMYALSKDKETGEHYVHYAYKHRQISGTGEEEQFHQLLPVESDDVLGI
LFSDQPYEYPDAWNRPFLRNGPEGFYIWFDPAYAEDTDVDEHFAREVRERLLKMKQAGKVDEEAVRKLFRELDDLK
>lcl|NZ_CP026520.1_prot_WP_042228046.1_2598 [locus_tag=PC41400_RS13240] [protein=stalk domain-containing protein] [protein_id=WP_042228046.1] [location=complement(2869773..2870447)] [gbkey=CDS]
MKKKLAATAIGAAMVLSMGTGVLAGANLQEIKAYLNGSITVKYNGSPVQLQDEQGAVVLPITYDGNTYLPVRSIAKTLNV
AVNYDGATNTVLLGEQAEGVALAKDFKNSSYHTKDPNLTTYGGKDYKEVYFDNAKSSRSSSFMLYPKGKYQKLVLQVAAT
GNDIESFYVQDSKKDIKLKTTTIAIQDGLKTIEVDIAGVEELFVHADIKEDGGMFVPLTTSYFK
>lcl|NZ_CP026520.1_prot_WP_164977125.1_2599 [locus_tag=PC41400_RS29780] [protein=hypothetical protein] [protein_id=WP_164977125.1] [location=2870779..2870937] [gbkey=CDS]
MKTNGYTFAELNGKEELIEEIRSLESKLQEQTGEPITLIAYEKERAGNGERA
>lcl|NZ_CP026520.1_prot_WP_042228045.1_2600 [locus_tag=PC41400_RS13245] [protein=MoxR family ATPase] [protein_id=WP_042228045.1] [location=2871121..2872032] [gbkey=CDS]
MNSTPADFMRQWELPESITNRLLTLRGRIESGQLAEEQLRLIGKPGFVPAQPEVLTDAIIALALGKPVLLKGPTGSGKTK
LAEHLSFLTGQPMHSVNCSTDLDAEALLGFKTLSRSGEGTTIEFVPGPVITAMKAGHILYIDEVNMAKPETLPLINGVLD
HRRTLTNPFTGETVTAHEDFRVIAAINEGYIGTVPLNEALKNRFVAVDVPYLQGDALLGLLTERTELRDTRLLEAFAALS
ADLLALVQMGQLSDEAASVRALLDACDLAAFMPPRRAVERAIAAKLEDERERAAVRNVAETLF
>lcl|NZ_CP026520.1_prot_WP_042228044.1_2601 [locus_tag=PC41400_RS13250] [protein=hypothetical protein] [protein_id=WP_042228044.1] [location=2872035..2874011] [gbkey=CDS]
MFVGKLNLLDNRLEAFVLMQLVDLARTFTRNGDLELQLAFHSHYEESKALLTVSQFWRDWKPAQREAGMKSDVYLRALGS
AWYSGRRETEAFLESVTESPLSRLGNELFALCEDLRLEQICRRIRPGTSAVFEDRADVYKGYFTHKFKTHLARHEFADAL
FVSVYLWFKGADVHALYSETGSRIGETFAESAAPTKHVEELLRLVQPLLEEAETAETTAENARLCLQMLELLEYRLERDA
AVAYFSAQPGSGTDAPLRRGERPAGKKAKRLANDDAERLSPRQDKDRRSERQSMWHRETRSTAGEALRQSVERGTRTARF
SGAAARPGDDTAEEATAAARGRSAASERKSEPPPDSSAPFIPSGASAGSGEGSSGLRAVAEFQAPRRPTPAEEDAYREAI
RAVEPLTRQLTRTVRLTLERKRVAPRSDLLYGRLGRRQLLRYVTDEAPRLFTKKQNPAPKLDAAFSLLVDCSASMYDKME
ETKLGISLVHETLRTLRIPHEVTGFWEDADRVTAEEAPNVFHTVIDFGSSSAPRSGAALLQLEPQQDNRDGFAIRHAARR
LLRRPEKQRVLLVFSDGEPSAAEYQEDGILDTYEAVLECRRLGIDVISVFLASGEVHETQRTTMRNIYGRHGIVVPHAAE
LPEHLAPLLRKLLLHSGG
>lcl|NZ_CP026520.1_prot_WP_042228042.1_2602 [gene=purT] [locus_tag=PC41400_RS13255] [protein=formate-dependent phosphoribosylglycinamide formyltransferase] [protein_id=WP_042228042.1] [location=2874150..2875331] [gbkey=CDS]
MFGAPLSGKSKKIMLLGSGELGKEVILEAQRLGIETIAVDRYDHAPAMQVAHRSYVIDMLDRQALREIIERERPHLIVPE
IEAIATQELVELEKEGYRVIPTANASRLTMDREGIRRLAAEELGLRTAKYAFADTLDELHAAVAEIGTPCVIKPIMSSSG
KGQSVCRSEADIEACWNTALEGGRAKKARVIVEAFVTFESEITLLTVRSVSGTSFCAPIGHVQKEGDYIESWQPHVMSES
HIAEAEQMARTITDALGGYGIFGVELFLTSEGVYFSEVSPRPHDTGMVTMATQDLSEFALHARAILGLPIPQIRLLTPGA
SHTLKADRGSTDFQIGGLEEALAVPDVQVRVFGKPETKPGRRMAVALAAAQDTETARAMAAEAAGKLFIRYGE
>lcl|NZ_CP026520.1_prot_WP_042228040.1_2603 [locus_tag=PC41400_RS13260] [protein=acyl-CoA thioesterase] [protein_id=WP_042228040.1] [location=2875429..2875908] [gbkey=CDS]
MEAKPTSVSRSIMTEVIFPLDTNYHGTVFGGKIMEYIDKIATIASMRHCRRGVVTASSDSLDFLAPVHLGEAIELEAFVS
WTHRSSMEVYVRVVAENMFTGERRTTATSFITFVALDDEGRPTPVPAVIPETEREKRLHESAPARYEARKKRKAEREAD
>lcl|NZ_CP026520.1_prot_WP_042228039.1_2604 [locus_tag=PC41400_RS13265] [protein=TIGR00266 family protein] [protein_id=WP_042228039.1] [location=2876035..2876817] [gbkey=CDS]
MNAHEIDYVIHGNEMQYVEIELDPQESVVAEAGSMMMMEGNIGMETIFGDGSGSGGRGLMGKLMGAGKRLLTGESLFMTV
YTNNGRGKEHVTFAAPYPGKIIPMDLHALGGKLICQKDSFLCAAKGVSIGIELQRKLGAGFFGGEGFVMQKLEGDGLGFV
HACGSILERELQPGEVLRVDTGCFVAMTPDVQYDIEFVKGIKTAVFGGEGLFFVTLRGPGKVWLQSLPFSRLAERVLGGA
IRGKEEGSVLGGLGNLFERH
>lcl|NZ_CP026520.1_prot_WP_042228037.1_2605 [locus_tag=PC41400_RS13270] [protein=hypothetical protein] [protein_id=WP_042228037.1] [location=complement(2877024..2877221)] [gbkey=CDS]
MNNNFTADAERVLIVYLNNDIVVENVPGEVDVDSYLDEQDYDARQVELISMGEFNERLDQMLLQY
>lcl|NZ_CP026520.1_prot_WP_042228035.1_2606 [locus_tag=PC41400_RS13275] [protein=sulfate ABC transporter substrate-binding protein] [protein_id=WP_042228035.1] [location=2877551..2878645] [gbkey=CDS]
MAALKSHSFKRVLGRSVSVLALTSLLTGLTACGGGNETTSTKTGTENGGSAPKSVELLNVSYDPTRELYQAYNEAFIKYW
KEKKGQTVTVSQSHGGSGKQARSVIDGLEADVVTLALAYDIDQVQAAGLIDKGWETELKDNSTPYTSTIVFLVRKGNPKG
IKDWDDLIKPGVSVITPNPKTSGGARWNYVAAWGYALKKNNNDEAKAKEFVTQLFKNAPVLGTGARDSTTTFVERGLGDV
LIAWENEAYLSLNELGKDKFEIVTPSISVLAEPPVAVVDKNADKKGTKEVAQAYLDYLYTEEGQKIAAKNYYRPRLESVA
KQYENQFKPVNLFKIDDPEFGGWAKVQKTHFADGGVFDQIYKPK
>lcl|NZ_CP026520.1_prot_WP_042228033.1_2607 [gene=cysT] [locus_tag=PC41400_RS13280] [protein=sulfate ABC transporter permease subunit CysT] [protein_id=WP_042228033.1] [location=2878715..2879548] [gbkey=CDS]
MGRRNKTKRILPGFGLSMGFTVLYLSLLVLFPLSAVVWKTTGLTFDQFWQAVTDERVVASYKLSFGASLAAALVNTVFGL
LVAWVLVRYRFPGKRFVDALVDMPFALPTAVAGIALTSLYADNGWIGSLLAPLGVKAAYSPLGIIIALTFIGLPFVVRTV
QPVLQELEHETEEAAVSLGASRWRTFRRVIFPQLVPALLAGFTLAFARAIGEYGSVVFISGNMPMKTEITPLLIMTKLEQ
YDYAGATAIALVMLIISFVLLLVTNALQMKSNRRVSG
>lcl|NZ_CP026520.1_prot_WP_042228031.1_2608 [gene=cysW] [locus_tag=PC41400_RS13285] [protein=sulfate ABC transporter permease subunit CysW] [protein_id=WP_042228031.1] [location=2879571..2880482] [gbkey=CDS]
MAGNVAANLNGPSSEPAKHITEPRVVQWLLIGGALLFLALLLVLPLATVFIQAFKKGTEVYLAALTQKEAWAAIRLTLTT
AAIAVPLNVVFGVAAAWAITKFKFRGKNVLVTLIDLPFAISPVIAGLMFILLFGTKGWFGPWLADHDLKIVFALPGIVLA
TVFITFPFVARELIPLMQAQGTQEEEAAATLGAKGWRIFWKVTLPNIKWGLLYGIILCNARAMGEFGAVSVVSGHIRGKT
NTVPLHIEILYNEYQFSAAFAVASLLVMLAVLTLIVKSLAEWKFHQSNRQHADSGLNQATEGS
>lcl|NZ_CP026520.1_prot_WP_042228029.1_2609 [gene=cysA] [locus_tag=PC41400_RS13290] [protein=sulfate ABC transporter ATP-binding protein] [protein_id=WP_042228029.1] [location=2880486..2881541] [gbkey=CDS]
MHISVRGINKSFGSYEAAKNISFDIEQGKLIGLLGPSGGGKTTILRMLAGLESPDSGEILFHGEKVNALMPQERHIGFVF
QNYALFKHMSVYDNIAFGLQVQKRGKAEIRTRVHELLQLTGLAGLEKRLPHQLSGGQRQRVAFARALAPEPQLLLLDEPF
AAIDAKVRKELRSWLREMISRVGITTIFVTHDQEEAVEVADEIMIVNKGRIEQKGSPIDIYREPLTPFVASFIGESSPLE
EAGSLIGFADAGEASAIVRPEFVEIGLPGEISRQTASAEAVVRSTYFRGSNWQVEVESGPYRLVGYRSLELPQLEAGDHV
HVLVHRLYIFDGNQTKTVENALKNDPMPVHI
>lcl|NZ_CP026520.1_prot_WP_042228027.1_2610 [locus_tag=PC41400_RS13295] [protein=alpha/beta fold hydrolase] [protein_id=WP_042228027.1] [location=2881584..2882417] [gbkey=CDS]
MEHHITIPSGDLELAATLHYPTDGEAADCRGGAGKWPVVIICHGFIGNRIGVDRLFVKTARALSSQGYMVLRFDYGGCGE
STGDYGAGGLDPMIAQTRRVLDYALSLDCVDPGSVTLLGHSLGGAVALLTAASDHRVRTLVMWSAVAHPHNDIVRIVGKE
AYRQLEPDKGIDYQGYLLTPAFFDSLSKHQPFEQLRKFGGDVLIAHGTADEDIPVDYAPLYQKMFWLRADGRCDLELIFQ
ADHTFSNRDATFTLIGKTSAWLVDLQKRKREWYDWMI
>lcl|NZ_CP026520.1_prot_WP_042228026.1_2611 [locus_tag=PC41400_RS13300] [protein=YezD family protein] [protein_id=WP_042228026.1] [location=2882443..2882664] [gbkey=CDS]
MAKPVELNEVWIDRIIQSINGLEYGTVQIIIHDGRVAQIERTERKRFDNETKRSAAGAEAKLHIERKASDGRG
>lcl|NZ_CP026520.1_prot_WP_042228025.1_2612 [locus_tag=PC41400_RS13305] [protein=NCS2 family permease] [protein_id=WP_042228025.1] [location=2882889..2884196] [gbkey=CDS]
MDKAEAWFKLKERRTTIGTEVLAGCTTFMTMAFVVIVNPGILSASGMDFHAVFIATILTTILSTLIIALAGNYPIVIAPG
MGINAFFAFGVVQGAGVSWQVALGSVFLAGLLFMLISLSRHGRLLIESIPASLKYAITAGTGLFICFVGLQNAKLIVASP
DTLVTIGSLHEPAALLSVIGLVLTLVLMSYRVRAALFIGMLLTAAIGWLMGLLILPDHWFAWPGTMTGTFMQMDVKGVFE
QGLFAVTFTFLLISVFDTTGTLVSLAEQAGLMKNGVFPRSRSALMANAAGVTAGAVMGTSPVTTLIESGSGIAAGGRTGL
TAVVTCALLAATSFLAPVAETLANLPSVTAPALIVVGYYMITDLLHIDWKDFEESFPAFLVLVTMPLTYSVSTSIGIGFI
FYIVLKAARGRFKEIHPVLYVFAALFIVQIGFFHG
>lcl|NZ_CP026520.1_prot_WP_042228024.1_2613 [locus_tag=PC41400_RS13310] [protein=alpha/beta hydrolase] [protein_id=WP_042228024.1] [location=2884235..2885188] [gbkey=CDS]
MKESHFHLNEETQTPVFVYRWEPEPGTPLRGAVQIAHGMAEHALRYKRFAEKLTARGYIVYANDHRGHGRTAAHKNDLGY
PGPDGFAGMTDDMIDLGRRIHEEQGGLPLYLVGHSMGSFLSQKVMYRAPQLYEGVALLGTNGPKGVILKAGVLLAKLLVT
RSGDCVRSKFLSHTALGSFNRPFRPNRTEFDWLSTDEAEVDKFVKDPLCGYVCTTGFFKDFFIHLSDIHLKRHMRSIPKH
LPVLLLAGDKDPVGGQGKGVRRLAALYRKLGLEQVDCKLYENKRHEILNEVNREEVMEDVISWLVSTGGLIKEPATS
>lcl|NZ_CP026520.1_prot_WP_042228021.1_2614 [locus_tag=PC41400_RS13315] [protein=hypothetical protein] [protein_id=WP_042228021.1] [location=2885313..2885804] [gbkey=CDS]
MVTGTVVGLLLSGFVAWKTYQARQAFKKVGVVVAKGHMYPLDFWVFAASMLVVVLLMGQDGLWFAALCGFIFYLEWVRLF
YLGTEGLKTGTKFYKRSRLITYEITEGNGPVIHIIVAGEMDPVRLSCKENDFAILKDQVIRYFEERPEGSRSADTGRGTI
STV
>lcl|NZ_CP026520.1_prot_WP_042228020.1_2615 [locus_tag=PC41400_RS13320] [protein=NAD-dependent succinate-semialdehyde dehydrogenase] [protein_id=WP_042228020.1] [location=complement(2885920..2887386)] [gbkey=CDS]
MTTTTSGSTSERWLNYVNGQWIEADSGKTISVVNPSTLGEIATVPDSGEAETKAAIEAAHTAFQSWSTLSAYERSDMLMR
WYHLLMRHQEDLARTMTAEQGKPLSEAKGEIAYAASFISWYAEEAKRLYGDIIPATSSGKRMFVLRQPVGVVAAITPWNF
PAAMITRKVGPALAAGCTCIVKPAEQTPLTALYMAKLAEEAGIPAGVVNIVTGEASVIGDTLFADSRVAKVTFTGSTEVG
KHIMRASADTMKKISLELGGHAPVIVLDDADAELAAEQTLLSKFRNAGQTCVCANRIYVQKGIRPAFEKLLADKIRALKV
GDGMNPDSAIGPVIDEDGLNKIQKHVSDAVERGATVITGGGRREVPERAGYFFEPTLLTGVDSSMLIMKEETFGPVAPIV
EFTDPDDAVRQANDSPYGLASYVFTSNINHAIRLAERLQYGIVGVNDGLPSSAQAPFGGMKESGLGREGGKYGIEEYLEI
KYISLGLY
>lcl|NZ_CP026520.1_prot_WP_042228018.1_2616 [locus_tag=PC41400_RS13325] [protein=aspartate ammonia-lyase] [protein_id=WP_042228018.1] [location=2887866..2889278] [gbkey=CDS]
MTEQQPSMEQNTKLRTAGDSLGQVQLPADAYYGAQTQRAVLNFPISGYRLPREFIKAQGIIKASAAAVHKELGELDGEIA
DAILAAADEVIEGRWDSQFVVDAFQAGAGTSQNMNANEVIANVALEKLGLERDRKDKIHPNDHVNKGQSTNDTIPTAICI
SVHESVERKLLPALDKLIGELANKSREFHLVVKSGRTHLQDAVPLRLGQEFEGYAATIGQMRKQLEQAADGLLELGIGGN
AVGTGVNTHPEYAARMVAEIGRRTGTPYRQAPSRFAFMQNPSAALRVSGLLRDLAVHLIKLSSDIRLLNSGPQTGLAEIR
LPAVQPGSSIMPGKVNPVLPEMLYMVCCQVIGNDTAVQTAMIGSQLEINVMMPVIAHNVLQSAHILSTAAEAFCERCVKG
IEADEEQCRYWMDRSLSLVTMLNPVIGYDKAAEIAKESLKTQKPLRQLVVEKGILTEDEWERIVQNSLSE
>lcl|NZ_CP026520.1_prot_WP_042228016.1_2617 [locus_tag=PC41400_RS13330] [protein=GNAT family N-acetyltransferase] [protein_id=WP_042228016.1] [location=complement(2889507..2890649)] [gbkey=CDS]
MNIRLVNQEEMREAAALADSIFRDSEQISMAEGFPFLFSALHSQSVGAFENGRLVAFMGLLPSVIRIGPARLTVYSLGSV
CTHPESRGKGIAGELFAAVRGQAARSGGSLLLVSGGRSLYTRAGCRPFGSLRRYALGSGAAQQAEESAGGYRLRPYERGD
LPAMSRLAAARRSAFEQSPWDLGSLIDAHAYAACLKLRHETWIAEKDGEPTAFIVAAVPDPAVASTRFPRLIEWAGEPEA
AARLLTGLTAKLGLAEFQAAVPFYETGLHEALRSFPSEESPNEGTLLVTDAPLLLRQLRPYLDEQSGNRLGERLGVRETG
EGGVRLTLNGLPDLLLSVDDWMALLFNPDAAVRIVPARWQEELKSVFPLPFPYTAGLNYI
>lcl|NZ_CP026520.1_prot_WP_042228014.1_2618 [locus_tag=PC41400_RS13335] [protein=ElyC/SanA/YdcF family protein] [protein_id=WP_042228014.1] [location=2890957..2891661] [gbkey=CDS]
MKLRWKGLAYGLGAAAIAGGIWLFTSDYEVRKEGEARKISAEQAPSAEAIIVLGAKVHPDGRLSDMLKDRMITAAELYEN
GKADKILVSGDHGTLGYNEVRAMKNFLLERGIAEERIFMDHAGFNTYESMYRAKAIFQIRKAIVVTQDYHLARALYDAEG
LGVEAYGVAADRQEYRGMAKFKAREMAARSKDFALIRFFKPEPTYLGEVIPITGDGRATEDSWLIEEAVWRPNA
>lcl|NZ_CP026520.1_prot_WP_042228012.1_2619 [locus_tag=PC41400_RS13340] [protein=ABC transporter substrate-binding protein] [protein_id=WP_042228012.1] [location=complement(2891723..2893591)] [gbkey=CDS]
MQITHHYLELCRAYSAVPQGLPVSLTLEELAERLHCTVRNVKLVIRKMSEAGWITWVPGRGRGNKSELHFHYTLEEMLMK
QTAEFAEKEQYQQAFEWIHQYADTPLMKEQLLQHVSRYFGYKAKVTEQDADVLRLPINKTYRTFDPAALLFANDVHLSRQ
VFDTIVRYDRAAGDFVPSLAHHWESDESCTVWTFYLRKGVLFHHGRELEAADAAYSIRRLLSPEVFSPQRWMLEDVRRIR
VLNRITIQFVLDKPHVLFPHCLTSPSAAILPRELCEADPAAYFACPVGTGPFRLEIRPGYGYLLHANERYYLGRPFLDVV
EHILLRDSPELVVMTDACNMVQRAPYAPRSKPEHDWQAVECISSGCALLSFNRRKEGPLQDAGLRRYVRSVIQACAMVEE
LGESRAIAAASFLPDKSLRIGCDPYRDEKETADIPYSSAYAGTELTLHCKDSYKTYEDIDWIVNRCLKAGLHIKVALHER
DEFLTPDVMDEADLVLYEVVLDESETLSLLELFMQEEGSIRRMFDDPLRAELNRRVALLLAEPEPRKRLDLFDKLESLLT
EDCSIHFLYHRRMQTTYHKSVRGLRLNAFGHIDYRHIWFEPQDPLTGQTASADTSSPFMTIH
>lcl|NZ_CP026520.1_prot_WP_042228010.1_2620 [locus_tag=PC41400_RS13345] [protein=glucose 1-dehydrogenase] [protein_id=WP_042228010.1] [location=2893699..2894451] [gbkey=CDS]
MKTEPAGKAVIVTGAASGIGRVIARRYAERGSAVVLADTDETRGEDNARTIREGGGEALFVPADMSRPGDIVRLIERAAA
AYGSIGCLINNAGVSDWKSPYDLTVEEWDRVLHTNLRGTFLCAREAAKRMRETGGGSIVNIASTRALMSEPSSEAYAASK
GGIAALTHALAISLGPDGIRVNCISPGWIETGDYEALKPSDHKQHPAGRVGTPDDVARACFYLTDPANDFVTGENLVIDG
GMTRKMMYEE
>lcl|NZ_CP026520.1_prot_WP_042228008.1_2621 [locus_tag=PC41400_RS13350] [protein=DUF1294 domain-containing protein] [protein_id=WP_042228008.1] [location=2894539..2894820] [gbkey=CDS]
MFSIKILVLYLLVVNLVAYVWMGADKRRARSGEWRVPEKRLFLTALIGGAPGIWAGMYAYKHKTRHFTFKYGIPLLLIVN
VIVFWKLFRFVGE
>lcl|NZ_CP026520.1_prot_WP_042228007.1_2622 [locus_tag=PC41400_RS13355] [protein=hypothetical protein] [protein_id=WP_042228007.1] [location=2895178..2895423] [gbkey=CDS]
MDHTFLIVKVAEQEQLHTVIRTLKQHMLAPLVAWKTPGKRHRSHLILRGDLARLTEAKLLLGGLMSAQSIEMDFQVLMLA
S
>lcl|NZ_CP026520.1_prot_WP_042228005.1_2623 [locus_tag=PC41400_RS13360] [protein=hypothetical protein] [protein_id=WP_042228005.1] [location=2895537..2895926] [gbkey=CDS]
MNPDFISIQMTDGELKYSHKKSGLGLTVTTKELIVQKPHVNYYIPLQAILSIVPFEAKGRTLTFSSRNGGGSELTSMSAG
LPHYRLSVSGLRLHNRSGIFTMGAVEFIVPLRDELLLTVSRLSGLHAVE
>lcl|NZ_CP026520.1_prot_WP_042228003.1_2624 [locus_tag=PC41400_RS13365] [protein=hypothetical protein] [protein_id=WP_042228003.1] [location=complement(2896111..2896323)] [gbkey=CDS]
MTDNHDPNFELGAPRSSLDSAEASSAHGTELTREQFLQSYEAGTIEDDGGLGEEAQAEAGYRPESSSDRA
>lcl|NZ_CP026520.1_prot_WP_042228001.1_2625 [locus_tag=PC41400_RS13370] [protein=aromatic acid exporter family protein] [protein_id=WP_042228001.1] [location=complement(2896426..2897481)] [gbkey=CDS]
MTFGARMLKTGMAVTLALYLSMLLHTTAPVIAAVAAIFALQPSIYRSWRYFLDQLQTNTLGAALALVAGTFFPKHPIAIG
IVCILAIMICLKMKMEANIGLTLVTVIAVMDASGGHWWFAVNRFLLTSLGVGSAFVINVLFFPPNWQKRFMEEVHGVFNQ
ISPLQRTVISNEMKESVFQETKDGVVGALFRLEEKYKLFEEEQKKLKRLKYSQSKRLVVYKQMLHTLRKGVDILQLVEEH
YFQASRTSKMNEFVDEHLEALTQYHEYVLMKFEKKYKNESYEKLEAMDLKNDAFLRLITDASSSTGDLHRLVVVASGIYD
YGHQIVRLDKLVEQFFKGAEENEKQTGLKFF
>lcl|NZ_CP026520.1_prot_WP_042227999.1_2626 [locus_tag=PC41400_RS13375] [protein=hypothetical protein] [protein_id=WP_042227999.1] [location=2897693..2898412] [gbkey=CDS]
MELLLIVIYLGIVLLVIEISAVLMMLTGLDNEIARFQVTSLLTGTGFTTQESELIIRHPVRRRIGIFLILFGVFSLAVII
TLVSNLLEKRFRLPELLIITGILVFLLFVLKLKPMQKRLSKKLHAPLEEDEGVEELPVSELLYNDEEDWIIDIPVIEKSE
FEHCHIDDLCTANEDINLLFVQRGPLRIRRRGENSDTRLEAGDVLYVYGNKKQISEKFSKELEHRHENLVDENKTVSLL
>lcl|NZ_CP026520.1_prot_WP_042227998.1_2627 [locus_tag=PC41400_RS13380] [protein=PepSY domain-containing protein] [protein_id=WP_042227998.1] [location=2898481..2900829] [gbkey=CDS]
MYDKRASRLRKAGMLTVAGSLGVGLLLPGITNAANDSAPAYRFQEDAKIVSGSGPAQPSESSDSAILEKEAKITKEQAIG
RLRELFPDLKNTEVSSASLGESTDRYPPVPRLVWTISWSVTKGNSTSSFHSEVDARTGDVWSTSFPGAWLKEEESFYPPA
VNQEKAAQIALELVKKAIPSLQNDTLESFLSYGSQSALFGPVLYDFTFRRKVNGLFVDFEGINVQVRGDGRVTGLHASLH
KSSYPSPTPKISEREAAAKYKKDFAVDLQYVKLQNGSGPKEWILGYVRNPLFYAPVDAQTGEYVFHGLYAAYGTSAYADV
PRTGASFKAHTGGPLSAEQAEELVKATGYVPEGSKLAQKSEQEGWRTPGQLVWSLNWKNENGRDTHAIVDTRTGQILELS
ENGIYPPWNQNPNVKPRSITSESADAKALQWFDELVPQASENWKLVKKNSAGKDNVSSKKNEPFTYQFQRFYNRIPVQDQ
QIILNLNGDGNLYTLQLPYEQEAIEEALKAVKPAAVTKEEASKRLQADGRMSLSYTMNGAFYTGGDDKERKLLLVYRQGS
EAESAPQVLDAASGKLRDLRPIPAVPKPSEGKDITGHWAQKELAELVRLQVLVPDADGNLNPDAVLTYGDWLTMFARGIR
PDSYYGENYRTAEAFYADVQPGSPYYRTVSLLTDINRLQPDPDRKLGVDEPLTREKLASALVNELRYEKLSSLLDKPLGK
LAVTDEGRITNKADVFLVMKLGLLQDIGCEFRPQDAVTRAQAAIVLMRLAGLQGKTDQPISY
>lcl|NZ_CP026520.1_prot_WP_053228671.1_2628 [locus_tag=PC41400_RS13385] [protein=acyltransferase family protein] [protein_id=WP_053228671.1] [location=complement(2901335..2902405)] [gbkey=CDS]
MRNPTAVYFTNVRFFLILLVVLVNLLLTLPPLPEAAQAVVTAIFTFHLPVFVLVTGYFAKKYWQTPEAAAGMLRIALQYL
FFQTLYSVLDLFWLHSPGIRHSFWLPYWMLWFLTAHFGWKLLLIPFARLKHPVAAAVLLGAAAGFLPAEAGAWLGLTRTL
VFFPFFLAGYYWREEREPRRRTAPQRRAIRAAAALLLLALPAFWHAAGAETAAQWLAGGSSYAQLGVPAAAGTMLRLGCY
AAQAAASAAFLLLLPRAESRLTVLGGRTVYVFLLHGAAIKLLGAAGLLQLLARLMDAGAAGTAAALAAGCAAAAVIVAAL
SGRRFAAVMRPCIEPDAARALRWLQRRLPGRPRSTY
>lcl|NZ_CP026520.1_prot_WP_042227995.1_2629 [locus_tag=PC41400_RS13390] [protein=hypothetical protein] [protein_id=WP_042227995.1] [location=2902978..2903184] [gbkey=CDS]
MTALHLVNAAPAHCQACGARLQTHPEGYKDLCFDCVQTWQTAKAQKAGKTGLPLPLEAECGLIEQYWL
>lcl|NZ_CP026520.1_prot_WP_042227993.1_2630 [locus_tag=PC41400_RS13395] [protein=DoxX family protein] [protein_id=WP_042227993.1] [location=2903244..2903783] [gbkey=CDS]
MFNWLRTNVYAAGLMVILRFYLGYAWITGGWHKLTGGFDATGFLKGAVTKPVMDKATNELVYPTFTAFLENFALPNAKLF
NVLIPLGEFLVGLGLVLGALTTAAAFFGVLMNFMFLFAGTVSTNPWMILLGVIVIAAGANAGKFGADYYILPYLRKLLRH
RGDHGANPTLGGGKTRAHI
>lcl|NZ_CP026520.1_prot_WP_042227991.1_2631 [locus_tag=PC41400_RS13400] [protein=hypothetical protein] [protein_id=WP_042227991.1] [location=2904185..2905057] [gbkey=CDS]
MDRRIIRIELFEFDVSCCYHHARGKLENCRYGLLKLTDEGGETGWGECLMSVGEKHFDLVNWARFLYRFQKSTLTQAFDT
VRLYRVLWGNVKADLVEMALYDLKARSQRKPFVDVLGSFSGLGSFGSFRSARHSGRSENGSFHLETAAARSVGGGPCPDT
ESPFSSRVRQTAGFGHTVADRVFPVSGLFQAGRELHPGCCGSLLDDLRNACMLRQEGMALAMHKDYLIGPACCLWEHTAA
VLGADWLEEGGNPEPARLASLQASERIAGTGFQLDQPVLFEKSRAYFVVI
>lcl|NZ_CP026520.1_prot_WP_042227990.1_2632 [locus_tag=PC41400_RS13405] [protein=L-lactate dehydrogenase] [protein_id=WP_042227990.1] [location=2905130..2906083] [gbkey=CDS]
MSTNELNNQETLKKSRIVVIGTGAVGSTTAYTLLLRERMSELVLIDYNHAKANGDALDMNHGLPFTGRVKVWAGDYSDCE
GADIIIICAGVAQKQGETRLDLLKRNVDIFNGIVENVTKYNQTGIILVATNPVDIMSYFTWKKSEWPVNRVIGSGTLLDS
ARFRYLIGEKLQVDPRSIHAAIIGEHGDSEVPLWSLANFAGLPLELNQEDKDEIFQNTRDAAYQIIEAKGSTYYAIALAL
DRICTAIMRDEGAVLNVSTFVSNYHGLSDVYLGVPCIVDRTGVREVVTLPLSEEEENKLRKSGAKIREMIDSISERL
>lcl|NZ_CP026520.1_prot_WP_042227989.1_2633 [locus_tag=PC41400_RS13410] [protein=DeoR/GlpR family DNA-binding transcription regulator] [protein_id=WP_042227989.1] [location=2906278..2907054] [gbkey=CDS]
MFQEERIEAIMEYLREHQRMDVKEICDRFDVSRDTARRDLVRMDEIGLIVRTRGGAMLPTLNRAVTAYKDRMQRATDEKR
QIGREAAKLIRDGHYLFLDTSTTVATVAEHIRTTGNVIVTNSIDIAAKLADNKSLKVHMLGGQLELEHRFLFGATALAQL
ADYYVDTLFLGACGISPDGLSYPHEEDGSIKREMIRRADRVVVLADHTKFGKQLFYKVAGLDDLDVIVTDREPEESMLRL
LERHDVELLIVEGDNQHD
>lcl|NZ_CP026520.1_prot_WP_042227987.1_2634 [locus_tag=PC41400_RS13415] [protein=HAD family hydrolase] [protein_id=WP_042227987.1] [location=2907047..2907877] [gbkey=CDS]
MIKLIVTDLDGTLMNADLEVADADARMLREAAGQGVQICIASGRLYPDIRMIMDRVGVKAYGISQNGSAVYTREGEQVDA
SHFPADRASDLFEIVSKREIVFFVCCSDETIRLHKSTDASRPYEERLLSPAVHQADIGREIRSGQVKPCKFSLFGDMEKL
LSFREELMEKYAGSVEAVISDHDCMDLMPAGISKGIGLARLLKELGLKHDEAACVGDSYNDLPMFALTPHSFAMIKSQDE
VKAGAGHTTSSVAEMVTAVLEHNRSRRGNEAVEAQV
>lcl|NZ_CP026520.1_prot_WP_042227986.1_2635 [locus_tag=PC41400_RS13420] [protein=cysteine dioxygenase family protein] [protein_id=WP_042227986.1] [location=2908325..2908816] [gbkey=CDS]
MNLLQLIEQHFGGLKKPTLNELRAGLEALSLTRDKISPYVTEPERLAYGRHVLLRTADVEVVVIHLPGHSKTAIHDHGSS
IGCARVIEGEMLNVWYELDRGGYPEPFMAAPVTGGECMYAPRGQIHEMRNEHEERIVSLHVYSPPLTGTKAYLPYREVLD
FVI
>lcl|NZ_CP026520.1_prot_WP_042227984.1_2636 [locus_tag=PC41400_RS13425] [protein=hypothetical protein] [protein_id=WP_042227984.1] [location=complement(2908998..2909648)] [gbkey=CDS]
MLDFILQYKWAFFIGGEAVFWGSIVCFLLLRYVWNYRKLSKVFILLFLLSDLWMATIGYLDYRRTGQFETFQLVILIFLA
YALFFGRGDLRKLDAFIAGKTAKWRGIPENELPPPELKLTGAAHTRRELRNFSVHFILYAAVTAVLIVTYGILPAGSLPP
EPSGPPALRLVLAGIVNHETAASVLKVWSLILAVDAAITLSYVIFPKKPKKTATPG
>lcl|NZ_CP026520.1_prot_WP_042227983.1_2637 [locus_tag=PC41400_RS13430] [protein=cation diffusion facilitator family transporter] [protein_id=WP_042227983.1] [location=complement(2909731..2910600)] [gbkey=CDS]
MEAYNDLKQGERGAWTSIVAYIALTVLKLGIGYRAGSEALTADGLNNATDLIASVAVLIGLRISRKPPDSDHPYGHFRAE
TIAALIASFIMFAVGIQVFIQAISSFTGPKLEAPGMLAAWVALLCAAIMYGVYIYNIRLARKINNKALLAAAYDNRSDAF
VSLAAFAGIIGARLGLPWLDPVAAVLVSLLICKTAWGIFIDAAHALTDGFDDKKLALYKQTIAGTPGVEAIEDVRARVHG
NQVLLDVVIVVDHQLTVAQSHDIAEDVEQRMLRKHKVGHVHVHVEPTPM
>lcl|NZ_CP026520.1_prot_WP_042227982.1_2638 [locus_tag=PC41400_RS13435] [protein=YCF48-related protein] [protein_id=WP_042227982.1] [location=2910972..2912204] [gbkey=CDS]
MIRRSTLLLIGLLLAGMLSGCLSTGSNEAKAPQAPEPTVTPEPQPTAKPGADTKKDTPSEFQIQTRLLDFHLIDERTGYA
WGVTNQSLRMYRTLDGGQTWQNVSPPSAGNFKFSPGYGQAISFLDKQRGWIVRSDLEGKQSTIWQTENGGGSWKAVDLPE
QMNPVVLDFQTPQNGWMLSITDPSKADSGRILYSTSNGGSTWQRAADAQLPQEGTITGLTFQNLRTGWLSMEVKGQPKMY
RSEDGGRTWKESKVIAAARSSEACEVVDVLQPVFFGSDRSKGWIPFSCPSSEADQFHGLFTDDGGVNWSYAPFKVGTDKE
SNAFTLPFFLSETDGWYLDKNIVYHTSDKGKTWAPLNEDKILSRTWSKYPQLTQLQFVSENVGWILLETKDQKRSVLVQT
KDGGKSWSVL
>lcl|NZ_CP026520.1_prot_WP_042227981.1_2639 [locus_tag=PC41400_RS13440] [protein=helix-turn-helix transcriptional regulator] [protein_id=WP_042227981.1] [location=complement(2912321..2912539)] [gbkey=CDS]
MAIIIHIDVMLAKRKMSVTELSERVGITMANLSILKNGKAKAVRFSTLEAICKTLDCQPGDILEYKSDEDTQ
>lcl|NZ_CP026520.1_prot_WP_042227980.1_2640 [locus_tag=PC41400_RS13445] [protein=DUF2975 domain-containing protein] [protein_id=WP_042227980.1] [location=complement(2912549..2913028)] [gbkey=CDS]
MKKASTLFLKLAVILMGIPVLAMCIYLVPNVANFASELYPTIASMKYLVFILMYGAAIPFYFALYQAFNLLRYMDKSKSF
SQLSVHALKNIKHCAITISSLYVLGMPLIHFIAKKVDPPIGLLGLIIIFSSLVIAFFAAVLQRLLQEAINIKSENDLTV
>lcl|NZ_CP026520.1_prot_WP_042227978.1_2641 [locus_tag=PC41400_RS13450] [protein=DUF2975 domain-containing protein] [protein_id=WP_042227978.1] [location=complement(2913045..2913518)] [gbkey=CDS]
MNVKRGSTHFLKVIIFLAGIAVLALCIYWLPEATRRDALANPDYHYQVYPFLVCAYGICITFSVAMYHVFKLLTYSERNN
AFSELSLKSLKVIKRCAFTVIFFILLVLVNLMVLAKTTGDDAAGPISLSLMGIFATSIVAVILEVLQKPIKNLLDKK
>lcl|NZ_CP026520.1_prot_WP_042227977.1_2642 [locus_tag=PC41400_RS13455] [protein=VOC family protein] [protein_id=WP_042227977.1] [location=complement(2913716..2914078)] [gbkey=CDS]
MIKQAGTIAVYVDDQQASLKFWTERAGFAEVANHPMGPDAAWIEVAPPGAQTRLVLYPKAIMPGWEEQKAAVVFECDDIQ
ETYKTLSAKGVRFLEAPNQMPWGTYARFEDIDGNEFLLKQ
>lcl|NZ_CP026520.1_prot_WP_042227976.1_2643 [locus_tag=PC41400_RS13460] [protein=YjgB family protein] [protein_id=WP_042227976.1] [location=2914343..2915008] [gbkey=CDS]
MNSNLTHKSRMLMMAAVMSLPLALSACGSESQPAPSPTAAPTVSAAPTPSASPSGSPEPSGAPVPTAKPGEGGSTGSDKA
AQVKGIMELARKGQIAGVEYAAHTGLFDDVEKKWGKPDKQEGAGKGQYATYSKRGVTFGFNKGMLIFDVRSYDKALQQLA
LADIKQTLGEPAEVTVNGDDTIYTYGAGSQYQLKFVIPKSTGKVDHISVYSPQDAKNLMAG
>lcl|NZ_CP026520.1_prot_WP_042227974.1_2644 [gene=mgrA] [locus_tag=PC41400_RS13465] [protein=L-glyceraldehyde 3-phosphate reductase] [protein_id=WP_042227974.1] [location=2915347..2916336] [gbkey=CDS]
MHYLADKDRYASMVYNRAGRSGLKLPVISLGLWHNFGGSDSYENGRAMIRKAFDLGITHFDLANNYGPPPGSAEENLGRI
LKTDLAPYRDELIISSKAGYLMWEGPYGEWGSRKNLTASLDQSLKRMGVDYVDIFYSHRFDPETPLEETMMALDSMVRQG
KALYVGVSNYTAEQTADAIRILKELGTPLLIHQPSYSMFNRWIENGLQDVLRDNGVGSIAYCPLAQGLLTTKYLGGIPAD
SRAAKPSVFLQEKDVTDEKLEQIRRLNELAGDRKQTLAQMALAWVLREGRLTSALIGASRPGQIEENAAVVANLDFSEEE
LALIESVLA
>lcl|NZ_CP026520.1_prot_WP_042227971.1_2645 [locus_tag=PC41400_RS13470] [protein=hypothetical protein] [protein_id=WP_042227971.1] [location=2916469..2916996] [gbkey=CDS]
MGRRFIIESVLVAIYGQLLVPGRPVEYILPYSTVAELYELKDSPEPVMPEPQEDAHVKEMIGELITFFEDPLNKKKIERA
LTVPWRKSPPILVNDMVSFTVVNSVDNAHYGESFDPVETELILTALHEKVPMLTDQVELVEKIIQAEIPVQIYDVDDFEF
AVEEGISADDFKNLL
>lcl|NZ_CP026520.1_prot_WP_042227970.1_2646 [gene=pdxK] [locus_tag=PC41400_RS13475] [protein=pyridoxine/pyridoxal/pyridoxamine kinase] [protein_id=WP_042227970.1] [location=2917221..2918033] [gbkey=CDS]
MYKALTIAGSDTSGGAGIQADLKTFQERGVYGMTALTTIVTMDPHNGWAHQVFPISTEVLAPQLETVLHGVGVDAMKTGM
LGSVEIIELVGRTIDRENVGNVVIDPVMVCKGNDEALHPETTVSLRDVLVPRSTIVTPNLFEAAQLSGHAPLRTVEDMKE
AAAAIKKLGAKFVLVKGGGKLQHDKAVDVLYDGETFELLESDRIETANTHGAGCTYSAAITAELAKGKSVREAVRTAKAF
ITEAIRHSFPLNQYVGPTLHSAYRVYGGGN
>lcl|NZ_CP026520.1_prot_WP_042227968.1_2647 [locus_tag=PC41400_RS13480] [protein=hypothetical protein] [protein_id=WP_042227968.1] [location=complement(2918133..2919269)] [gbkey=CDS]
MDLFYFLAIVVIGLLVLIPVNLKLFRKSAQSQQPEKPAEEPASGSRVNNLTDGHAPETAGPHGEVPSVHAGAEAAAPAAA
PNLADAARAGTHPESLEEDRAPEVTVPGASSREPDPAPARPGTAAASAPAPVEQEEARFAPAAEETASSAQPEAEPELPP
RSQRHAGHTHGRKASLRGSSIRPGAQTPRAEAPLPAEPPHEAAEPSAAGALSEDAAGISAAAAAASAGEAPRLQLHPFAD
RKRSPEPPHVEDAASLRADEAPAEHPAPSAPAAAAGSTPAADRAPSGVPVPLAAEMGASAHAAPAARSGSAGEDGGVHSD
VDYREALRRGLRPEPEPEIVPEPRKDEKYDDRKYRDTLRDMFREAAKNAGNGDTDKRK
>lcl|NZ_CP026520.1_prot_WP_042227966.1_2648 [locus_tag=PC41400_RS13485] [protein=NAD(P)H-dependent oxidoreductase] [protein_id=WP_042227966.1] [location=complement(2919295..2919831)] [gbkey=CDS]
MKIFVVYDSESGHTEALARSIAEGAAAHPEAEVLLRHVNEAQISELEDVDAIIWGCPGHFGTISSGLKSWIDKLGYMWAK
GELIGKVGAAFCTTATVHGGIEMTLWNLITPMLHNGMIIVGLPSNVPENVLYGSYYGVGITCPVEASKDAPPNLPTESDL
ALGRALGKRVADIAQKLA
>lcl|NZ_CP026520.1_prot_WP_042227964.1_2649 [locus_tag=PC41400_RS13490] [protein=DHA2 family efflux MFS transporter permease subunit] [protein_id=WP_042227964.1] [location=complement(2919877..2921526)] [gbkey=CDS]
MSETKALVVYKEKEPEEDMTKHRGTVLLVLILGAFVTILNQTLLNVALPHLMNDFNASADTIQWLSTGYMLINGIIIPIS
AFLIGTFTTRQLFISAMISFTVGSLICGMAPNFTIMMIGRIVQGAGAGIIMPLMMTVILNIYPPQVRGRAMGMIGIAMFF
APAIGPTLSGWIIEHWTWRILFYVAMPIAILDIIFAFLVLKNVTKVTKPVFDFLGFVTSTIGFGSLLYGISEAGSKGWGS
VTVVSTIVIGIVFIALFIVRELTAKNPLLNLRVFKYNMFSLSTVVSCIVNMAMFGASILVPIYVQNIRGYSPIEAGLLML
PGALLMGVMSPISGALFDRFGIKPLAFIGLTITAITTWQFAHLNADTTYSTLMWLYTIRMFGMSFIAMTIMTTGLNQLPR
ELASHGTAASNTARQIASSIGTAVLVTIMTNRTTYHVGQYANDLSLSNPNYNPSGLESMGAAFAQQMGQSLAAGKAYASQ
MLYGLVTKQSTIQGINDAFVVATGLTIIALVVSFFFRRVAHGKSAKKKLAEKPAEPIELTDSRGQTARV
>lcl|NZ_CP026520.1_prot_WP_009672433.1_2650 [locus_tag=PC41400_RS13495] [protein=efflux RND transporter periplasmic adaptor subunit] [protein_id=WP_009672433.1] [location=complement(2921541..2922191)] [gbkey=CDS]
MGYTRIVITNVVVIIVLIVAAVTGYYFYNQSTTYIKTENAQIAGQQMLITAPATGKLEDWKGKVGQVFKAGDVIGTVQTA
GAQGTTAVNVSMPADGTVVQTSAYDNETVAPGTPLGYAYNMGSLFVNANIEETQLKNVKVGQTVDIYVDAFPNNTLTGTV
NQIGQATAGTFSMLPQSNSNANYTKVTQVVPVKIMLTDNQGVALVPGMSAEVRIHK
>lcl|NZ_CP026520.1_prot_WP_042227963.1_2651 [locus_tag=PC41400_RS13500] [protein=MarR family transcriptional regulator] [protein_id=WP_042227963.1] [location=complement(2922198..2922638)] [gbkey=CDS]
MESPLYPIYTSFREVNQTIKRILRTNAEAMGLSHVQFTILRLLSNCPGTNVTELAERMQQTNSSVSVAIEQLVGMGLVNR
IRSKEDRRNVKLNLTDRGTEKVRLSMNEDSHLYKQLVRILEVPPQEIQNMLDTHQKILDRLNAKEE
>lcl|NZ_CP026520.1_prot_WP_042227960.1_2652 [locus_tag=PC41400_RS13505] [protein=LTA synthase family protein] [protein_id=WP_042227960.1] [location=complement(2922879..2924756)] [gbkey=CDS]
MSIRWKEWYKSMFVWTIVLLLAKMALFRILVYDKLTLDKILIDAFDVVVIACLIELFRPGRVKTALFWIFNLLVSLALLA
SAIYFQNFGTVPIYTTLKQLHQVGQVSSSVRSIMRPEDLLFFADILIVLLFKLVVRKKPKRNYVPFRLPSPVTYALMIVS
VLLASSSIAAASGTKNELKKGEELGFFAYQAVSLFDGLRANRTNYADISEVKQNLTELEASFPRSAPLAPGAAPNYFGTE
KGKNVIVVQLEAFQNFPINLEVNGQAITPNMNELLKSSFYFNNFFQQVGQGNTSDAEFISNTSIYPTATVAMSSGFNNRE
LPGLPRALNALQYETNTFHVNDVKFWDRDKMYPALGFKQYFDKPSFTNDNFNAFGASDEELYKTGMKRLVELKNENQPFY
AQFISASSHHPFKIPADKQTLDLPENIKGKQLGDYLEAVHYTDYALGTLINQLKENGMWDNTVLVLYGDHFGLQPQDNDP
KMVSEALGIDYHEQISRYNIPLIIHAPSQKEGKQIETAGGQLDIMPTVANLLGIRLDETGIVPFGHDLLNVTKNVSGIRY
YLPTGSFVNNDILFVPGEGFDDGTAVSLKTHKPVADFSQYRKDYDYIMDLMQLSDEYVKLLPVRK
>lcl|NZ_CP026520.1_prot_WP_042227959.1_2653 [locus_tag=PC41400_RS13510] [protein=SOS response-associated peptidase] [protein_id=WP_042227959.1] [location=2925074..2925748] [gbkey=CDS]
MCKQYSLTADLAELQNRFQVNKIAVPYKARYNISPRQHVPVIVSVNGELVLDEYRWGFVPFWARDAVNADGMSVHQDPAY
RKIFMRQRCIVPCSGFYYWKTEGKKSFPVRVVPRNREVFGIAGLYDVWSDPRGKELRTCTLLMTESNSLITSFHNQMPVI
LNQHSIGEWMSQGAIDTDRLIPLLKPFPAEAMEAYPVTPAISNLELDESHCIEEMNLKVAWIKE
>lcl|NZ_CP026520.1_prot_WP_042227958.1_2654 [locus_tag=PC41400_RS13515] [protein=MFS transporter] [protein_id=WP_042227958.1] [location=complement(2925852..2927192)] [gbkey=CDS]
MEAKKKWDLISVASIPLIMTLGNSMLIPILPLIREKLHVTSFQVSLLITVYSIAAIILIPVAGFLSDRYGRKIVIIPSLI
IAGIGGAVCGIAAWFMPNPYMTILLGRLLQGIGAAGAAPVAMPLVGDMFVKESDVSSGLGIIETANTGGKVLSPILGSAL
AALVWFLPFMAIPVFSLTSVLMVLFLVKKPQKSGNKVAGPDQNRSKAVSKAADKNGKPQANSPKESGNTDGTPISLPVFL
KDLKWIFSRKGRWLYAIFAIGCICMFLLFGVLFYLSELLEEKYNLHGILKGFVLAIPLTLLCLTSFITGKIIGQHKKRMK
WLTFIGSLLSLGATLGCLPFNSLYGLLTMLSLLGIGIGLALPCLDAFITAGIDKAQRGTISSFYSSMRFIGVAVGPPAVS
VLLSRYPTILFYVLGAACLLAAILALFAIHPSTDEEPRAKPEPSPA
>lcl|NZ_CP026520.1_prot_PC41400_RS13520_2655 [gene=gluQRS] [locus_tag=PC41400_RS13520] [protein=tRNA glutamyl-Q(34) synthetase GluQRS] [pseudo=true] [location=complement(2927347..2928316)] [gbkey=CDS]
MSSNIVRGRFAPTPSGELHIGNARTALLAWLQTRAAGGTFVLRMEDIDKPRCRPELAEQITEDLRWLGLHWDEGPEAGDP
FGPTCRAPEGTSTRRRSRRCVNAGICIPATAAGPSFTPSPALRTASPPKARPIPARAAS*RRPSVRRARRSKRPRSASRC
LRKTSLSAIWPRERRFFPAGPEGTSSCSGRTASWPTSLPPPSTMRPWASPTSCAAGICSTPPRAS*CCCAPWVANRRATR
TSRCSTGPTASVCPSGTAPCPCPPSAARESRPRHCSASSPVGLDRPARTGAGCRADGGFPARGIPAASRAAKPAAPSAAF
RGL
>lcl|NZ_CP026520.1_prot_WP_042227956.1_2656 [locus_tag=PC41400_RS13525] [protein=EcsC family protein] [protein_id=WP_042227956.1] [location=complement(2928362..2929180)] [gbkey=CDS]
MHPTETKDELLMRLNEIDKWEQEQKDIFFWEKLGRLPFALLDKVTPRFLQDRIGQLIDEIGSYIQTGGKYLVSEKGMLEH
LSKHTAEEKNFRQDVPSIREAAHLPLAVMDRAADSLAESRSALATVQGATTGFGGIFTLAIDIPALLGLSLKVLQEIAVC
YGYDPNDKKERIFIINCLQFSSSDIVGKKAILEKLSAYDAEGRQSVSQLQGWREVVTTYRDNFGWKKLFQMIPIAGMLFG
AYFNRGTIRDVAEAGKMLYRKRRILEKLNDAA
>lcl|NZ_CP026520.1_prot_WP_042227954.1_2657 [locus_tag=PC41400_RS13530] [protein=MFS transporter] [protein_id=WP_042227954.1] [location=2929357..2930640] [gbkey=CDS]
MRFRDFHPNVKIRITMHFVTGILSGMVLPFMAVYFAKTLGTTTAGLAVMGNIIAGVAATFIGGYYADRIGRKKLMLISEL
VCTAAYAVMAVANSPWYVSPYLTLAMSLVIGAAWGLSKPAADAMLIDVTEPGARKFMYRITYWLNNLSISIAGIIGAFLF
SSYLFELLLGASVVTFISYLVTLIWLSETLEPAAGHVEQGGVPKTGGPFHILKSYGEVFRDRTFMIYVLASLLMVSVEFN
LTEYIGIRLERDIHGAVLLPGMTGLNGLELLGILRTENTFAIVVLSLFTGYLLKKYGGTRTMFVGLFLNIAGYSYLAMGN
HPWALLVLMLIATIGELIYVPIKQAFLVNIVPDHARSSYMAVNGMLYSGAHFICAINVVIGGFLPAWGMALVLFATGLTG
MLLLGSILPKLVPTPNTTAKSNVASAL
>lcl|NZ_CP026520.1_prot_WP_042227952.1_2658 [gene=hrpB] [locus_tag=PC41400_RS13535] [protein=ATP-dependent helicase HrpB] [protein_id=WP_042227952.1] [location=2931243..2933810] [gbkey=CDS]
MNVPVEAVLPELLDVLRRERNAVLIAPPGAGKTTRVPLALLKEPWLDGKRILMLEPRRIAARAAAAYMASLLGEKTGETV
GYRMRSDTKVGPKTRIEVITEGVLTRMLQRDPSLEGAGIVLFDEFHERNLAADLGLALCLQSQSLLREDLKLLVMSATLE
GDAVAALMNGAPVVTSEGRMYPVETRYRREAVKGPVAAAVLDTVLQAWREEEGDLLVFLPGAPEIRRAEKELAARLPAGQ
ALVAPLYGSLTQEQQDRAILPDTQGRRKIVLATTIAETSLTVEGVRIVVDSGLVRVPRFSPRTGMSRLETTAVSRASADQ
RRGRAGRLAPGVCYRLWTEQEERRLAPAGRPEIAEADLAPLALDLAVWGARPDELAWLDAPPAAAYAQACSLLRRLGALD
EAGAVTAHGRKVAALGLHPRLAHMALTALPLGAGGLACELAALLGERDILRGTPSARDADLRPRAEALRAYSGPGRGTKR
PPHDSGAAGTAAAEPASFADVSAPGSADVSVLRRLGSDAARWKRELGVPEDESGSTERTGLLLAFAFPDRIAGQRSPGRF
TLSSGRGAVLPDMQLLSSAPWLVCADLDDQGAESTIYLGAPVGRDELEEHFAHEIVEEITVRWDRDAQAVRARKRTSLGM
LTLRESTIAAPDPELIKDALLQGIRSQGLEMLGWSKQARQLRERLSFMAHHDARWPDVSDEALLAALEEWLGPHVYGMKS
RGDLQRLNLAAVLEAMLTWEQRRLLDEQAPTHLTVPSGSRIPVDYSTPEAPALYVRLQEMFGLQATPAIGGGRVPLTLHL
LSPAQRPVQVTRDLESFWKEAYFEVKKDLKGRYPKHYWPDDPLEAIPTSRAKPRK
>lcl|NZ_CP026520.1_prot_WP_241688859.1_2659 [locus_tag=PC41400_RS13540] [protein=hypothetical protein] [protein_id=WP_241688859.1] [location=2933841..2934074] [gbkey=CDS]
MNETAIIRLIRHDELDGLLELYRQLHTEDSVIPPSEDLNHQWNEMFHNPKMKFVVTEDGGKLAAVWGISGTDRPEVS
>lcl|NZ_CP026520.1_prot_WP_241688896.1_2660 [gene=modA] [locus_tag=PC41400_RS13545] [protein=molybdate ABC transporter substrate-binding protein] [protein_id=WP_241688896.1] [location=2934127..2934951] [gbkey=CDS]
MKRVGQAIAAGVLLLGLAACSAGQTGSQGGKPDSSREENGKAAELTVSAAASLKDALEEIGRRYESANSGVKLLYNFGSS
GTLQKQIEQGAPADLFISAGVPQMKALQDKQLIDKDHSGVLLTNELVLIVPGDSPASIAGPEDLLKEPVKKIAVGEPKTV
PAGSYTQEVLNGAGIWDKLQPKIVWTKDVRQVLAYVESGNADAGFVYKTDALASSKVKTVYTAKEAAGISSIEYPAGIVS
GTKQTVEARKFYEYLSGAEARDIFTRYGFGLKSP
>lcl|NZ_CP026520.1_prot_WP_042227949.1_2661 [gene=modB] [locus_tag=PC41400_RS13550] [protein=molybdate ABC transporter permease subunit] [protein_id=WP_042227949.1] [location=2934955..2935644] [gbkey=CDS]
MPVNMEMFLSPVWLSLKIALLSGMLAFVFGVLAARLVHKRPFRGRTLLETVFMLPLVLPPTVIGFLLLLAFGRRSWIGRL
AEWLWDQPVVFSWGAGVIAATVVAFPLVYQSAKTGFQTIDPDLEGAARSMGAGERQVLFHVTLPLAGPALATAFVLGFAR
ALGEFGATIMIAGNIPGRTQTVPTAIYTAVDSGDYALAWCWTGAIVAFSFLLLLAGNFRKNPPAGNSAP
>lcl|NZ_CP026520.1_prot_WP_042227947.1_2662 [locus_tag=PC41400_RS13555] [protein=DUF3817 domain-containing protein] [protein_id=WP_042227947.1] [location=complement(2935700..2935996)] [gbkey=CDS]
MLSTPIGRLRVIGFLEGISFLLLLFIAMPLKHFADVPQPVTVVGGLHGLLFVLFLLAVAHVTFVHRWSILKVLTAILASF
LPFGTFVLDARLKKDGWS
>lcl|NZ_CP026520.1_prot_WP_042227946.1_2663 [gene=helD] [locus_tag=PC41400_RS13560] [protein=RNA polymerase recycling motor HelD] [protein_id=WP_042227946.1] [location=complement(2936141..2938579)] [gbkey=CDS]
MSVSKEEWQHEQQRIDLVRTKLRKQLSKIEPDVGQLKSQVVDIRKHFWDEVTVNLSHDDDILDTYLSLRQQAEVLSERER
THRQLKRQADRMNRLLPAPYFGRIDFKEDGANPAERVYVGVSSFLDEDGETFLVYDWRTPIASLYYDYAPGPAEYNTPGG
LVEGEIELKRQFIIRDGHIHFLFDTGLMIGDELLQQVLSKGAGSQMKTIVATIQKEQNRIIRNDSSRMLIVQGAAGSGKT
SAALQRVAYLLYKHRKTLTADQMVLFSPNPLFNSYVSTVLPELGEENMQQTTFQDYLHHRLGRRFNIEDPFDQIEYVLAD
EKTPGYAARVSGIRYKASTRFLHVIQAYKDQLLKEHMLFKDIIFRDQVIVSRQALSDRFYGYDPAIRLANRIELMKEWLL
EELAAYQEQEMQEDWVRDELDFLDKEQYQRAYNQLRKRRGRAESFDDADAEEQLLRKMIVREHFKPLRSRVKRLRFVDVP
GLYRQLYENKLLFQSLHRTAAAADTPVPAENADDRDGLPARWDEICAYTLERMNRGDLPYEDATPFLYLQELTEGFRTNT
IVRHVLIDEAQDYSPFQFEFLKRLFPRCRMTVLGDFNQAVFAQATTLEGSTPVAGLYGPGETEQIVLHRSYRSTREIVEF
TRGMLKDGAQIEPFDREGGKPVVFTSAGAGAGKGTDANTCADALVRRMTAEIKALQADGSTSIGVICKTAAESERAFEAL
NGLDELRLIRKDTPSFEKGVMVIPAYLAKGVEFDAVLIYDASAVAYSRESERKLFYTACTRAMHKLQLYSAGTLTPFVAE
LDPAVYTVETVE
>lcl|NZ_CP026520.1_prot_WP_241688860.1_2664 [locus_tag=PC41400_RS13565] [protein=serine hydrolase domain-containing protein] [protein_id=WP_241688860.1] [location=2938943..2940475] [gbkey=CDS]
MRMNLGAITKLTRSEKVRMRIGARTLLAAAAALLIAVPVPAKAAGSVSGKPDTTARIDAYMEAAMNRLHIPGAALGIVKE
GGDTFYLQGYGTSGPSGAKTTPQTAFVLGSTSKSITAAAVMQLAEEGKIDLEAPVRRYLPDFRTADPERSDRIRVKDLLY
QTSGFSKYDGVAALTQGDSTIGAHIGGLNKISLVRDPGTGFEYSNLNYNVLGGLVEKVSGTSYTAYVKDHLFEPLEMKSS
FSSPEEAGGADMADMAAGYQSVFGFKVPTRQLAHTGTVPSGYLISTAEDMTHYLSAQMNGGRYKEKSVWSEKSNAALHEP
EASMGGGASYAMGWTVKNGVLFHDGATENTYSFMVIHGDTGIILLLNTMDYLVPYDQIVMGIYGILQGEDAPVESLPNYI
RTYALADMALTAVLVLAVRSFYSLFRRKKRVKPTAGRIALSAAGIAGIHLLLPACILLLPKLLHAPWPVLMVFLPGLSQG
LYFMAILLVCTGLLKTAAVVRAIRTRKTAL
>lcl|NZ_CP026520.1_prot_WP_042227942.1_2665 [locus_tag=PC41400_RS13570] [protein=class I SAM-dependent methyltransferase] [protein_id=WP_042227942.1] [location=2940604..2941326] [gbkey=CDS]
MNSARLQRIRREEKAYHEQYFEEHDLYEKDSWLEEPDERVLEYLEMLQDGKPLRMLDLGCGVGRNSIPLARKLQPTGGKV
VCVDLLDKALEQLEANGGKYGVKEVIETGRADISEYEVPGNTFDYILAASSLEHVRSESALDKVLAGLAAGTKDGGINFI
LMNTNIREVDAETGEGRETLIELILTKEQALEKLRQAYAGWEELEVSHKPLDIEINRGETPVIMKTDCLSYAVRKREAGR
>lcl|NZ_CP026520.1_prot_WP_042227940.1_2666 [locus_tag=PC41400_RS13575] [protein=flavodoxin] [protein_id=WP_042227940.1] [location=2941549..2941998] [gbkey=CDS]
MSTFIVVYASLTGNTEEMADAVVEGIKQAGAAVTVKDILDARAAELEAYDGIVLGAYTWGDGELPDEFLDFYEEMDELKL
NGKIGAVFGSCDSSYAQYGAAVDLLTEKLQSLGCAMQEGLKIELCPSAPEREACREFGRQAVLRLQGAL
>lcl|NZ_CP026520.1_prot_WP_042227937.1_2667 [locus_tag=PC41400_RS13580] [protein=hypothetical protein] [protein_id=WP_042227937.1] [location=2941995..2942666] [gbkey=CDS]
MSGAPAASRMEDGRLEQFIRCPGCFYSGGNCQESEPWRESARTSADRIIREFYGIDASRRTADIILERLEMYLIGPEESG
LVMEDYLRMKKNLAKYLILALFQEEKTDPPLFLFETLSVQVEELDLELVMHLHVGEWANGSYTLKRYYLDEDWSVVKAYL
QLSLVFAYKAFGRLPGQIEAYNLLTGEAYRIRPKSDDLQASMDFLRLTDTLMREHKRGRIMLA
>lcl|NZ_CP026520.1_prot_WP_042227935.1_2668 [locus_tag=PC41400_RS13585] [protein=GNAT family N-acetyltransferase] [protein_id=WP_042227935.1] [location=2942846..2943391] [gbkey=CDS]
MIIAERPFNVNGLSYTIRSAALRDAEQLSRLRVQIDGETENMDREQGEAFIDTPGFERLIQADSDSPRNLFLVAAVEDRI
VAFSRCEGNQLIRFSHKVEFGVGVLKEFWGRGIGKNLLKTSIDWADSNGIKKMTLTVLETNEKAAQLYRKLGFEIEGVLK
KDKILSDGSYYNTLMMGRFNE
>lcl|NZ_CP026520.1_prot_WP_042227934.1_2669 [locus_tag=PC41400_RS13590] [protein=SGNH/GDSL hydrolase family protein] [protein_id=WP_042227934.1] [location=2943626..2944741] [gbkey=CDS]
MEVEVKQAGKIDIARYDKNLQTDTMADEAAVWHDPRRAPFTVTGFPWLKTEGRYRRLPVRPDRKLPEAVDRLADCTAGGQ
IRFSSDTAQLSVRVRLGGHAAMPHMTALGQCGFDCYIGEPGNQRYVATASFKPGDSEYTSRLYRFGEKRMRHITLNFPLY
QGVEEVWIGTDEDARVAEPLPYESPGRIIIYGTSITQGGCANRPGMSYPGILSRQINREFINLGFSGSGKGEPEVAEVIR
EIGNPALLVLDYEANTGEPESIRATLPVFIDLYREKHPEVPILVLSCIEFAAAGFDPAVRKKLDDRRVIQRQTVEDRIAK
GDSFITFFDGSALLGEDGDECTVDGIHPTDLGFKRMADGLLQVFNRCLQAE
>lcl|NZ_CP026520.1_prot_WP_241688861.1_2670 [locus_tag=PC41400_RS13595] [protein=extracellular solute-binding protein] [protein_id=WP_241688861.1] [location=2944723..2946315] [gbkey=CDS]
MLAGRVRQRMFISSRLGNRSLRPLALLRSSLLLGALAAASAGCEHSPETAGGGFIPTGSASPADSISIVTNAIGMSFPQG
LDENNNPYLRYIEKNTGLNIKVTLPPQDVYPEKLNVIMASGNTPDLINYSDKVWMNDNARQNKLMPLDELIDRYGKDLKE
KIPKEAWDQVTVDGKIYAVPSLNEVKGLELMYVRKDWLDRLGLAPPVTLDDYYKVIRAFTLDDPDGNGINDTIGLTFGES
LVRSAPFFGAFGVQLDQWADRYGELAYTSTLPETKEALAFLRKLYREKLLDPEFPLNRNPSMMDKIKNGKVGLFSAAWYD
TRGPIAANMQKDPRAVWIPLEFPRGPEGHSGVYATNLVRSYNVIPVGSANPEGVIRYLNFIAGDGYKTLKLGFENEIWSM
QNGKMVINFEEHNKHLYRGIYQAMEDVQDKETSKIRLDALGDFHLYDNLKRIEGHLIPNAFYGSPTPSMSKYYNIKGKEL
MEHFVRIIVGVEPLDSFEMAVERWRKEGGDDMTREVNEWYKKQGIPQGAR
>lcl|NZ_CP026520.1_prot_WP_042227932.1_2671 [locus_tag=PC41400_RS13600] [protein=sensor histidine kinase] [protein_id=WP_042227932.1] [location=2946312..2948153] [gbkey=CDS]
MRKTLRWLRDKFQRNIQVRLTCYFLLILLPLVLVSLFANERSQQILVEQAAERSRLALNSAMDYIDLTLQSVEELSTLIS
TDSELIGTLSRNGRELTPRYVMDFSQMMKQLANMNSINHIAAQISIYHYPSTRMFSTNYGPRDLDKSQQEFMHSMTLRNG
TGILYMLPGEIIYGSTTFGELMNSDSVTLVRSMDLYNPLREQNVLLVTLNKTRLLNLIKPLLPSPQTKIYLTNANNNLIA
GISQEGETRDKYLPSEPDMLSVTITSNTSNWKLTMMQPKSELYSKTEQVKLYTYGIITISILLALWISWIVYTGIASPLQ
KLSHGIKRLGSGKLSIQLENRRRDEFGYLTEAFNQMAQQQKHLIEDHYEQELRLAKTELKFLQSQINPHFLYNTLDSIYW
TAKNYEADEISEMVLNLSRFFRLSLNKGREVFSVEESFAHLHYYIRIQQIRFLENFEVQYRIEEATKGVPLLKLLLQPLV
ENAILHGMDGKEDGGRLVVSSRLEEGFLVLEVKDNGSGIPQERLAYIQEELGKLHRRKGKLYSLDEEGVDDLFGLRNVVS
RIKLYYGGSAELSIDSAEGGGTCTALKLPLVRCREDFSPGADSMIDYKEETGT
>lcl|NZ_CP026520.1_prot_WP_042227930.1_2672 [locus_tag=PC41400_RS13605] [protein=response regulator] [protein_id=WP_042227930.1] [location=2948150..2949748] [gbkey=CDS]
MNVMIVEDEVRLRTSLVNNIPWEENGIEVIAQAASGTEALFLFERKKPDIVLLDVQMPEMDGLTLARKLREKDAFVKMII
LSGHDNFGFAQKAIEAGVSQYLLKPAGDGEILEAVLSAAEQLRQELERWHSQAELQEKWSEHLPQLQSAFFQNLIGGKYA
LWEIRKISRDYLLDLDESMQYAVAVADVDPLGPGEPRYKSGDMPLLRFSLQCVAREFLQPHSCWVNTDASGYTLVLFAMP
ADDSPNDVMLRVHAAVVKLLSQAKECLKVTCSAGISGGTGGLTEISKLYAQAYQALQDRIVYGADIAVPYRDKPAAGSSV
PLSPGLEKTLEIALETGDEDKAMDTLSELWDQAMEGTETAAGMHEALMYWTSLMIRLIHKQGWSVRDVAKDDYPYYRDVT
MLSTKQQIRAWLERSVLHYTAYLQEQRKTTSHGVVKTILSIVEKEIESDITLHSVADRLYVNSSYLSRLFKQEMGVAFST
YVLERKMDRAKAALLEGAKVYDAARLVGYRDVSYFTKVFRKYWGVTPGEMKP
>lcl|NZ_CP026520.1_prot_WP_042227929.1_2673 [locus_tag=PC41400_RS13610] [protein=hypothetical protein] [protein_id=WP_042227929.1] [location=2949901..2950155] [gbkey=CDS]
MNTLLHADRKKRAWNELIPAKERKRVYAAALLTGVLVLAGMLYLLQQSGYHTRGFKGIFKNYGSLARLVLFFVLANYVLL
HAFI
>lcl|NZ_CP026520.1_prot_WP_042227927.1_2674 [locus_tag=PC41400_RS13615] [protein=ABC transporter permease subunit] [protein_id=WP_042227927.1] [location=2950358..2951314] [gbkey=CDS]
MRQPASEVNVSRSKQSYSRQLGSDIRKDWDLYLLLVPGILFVLLFKYTPMYGLVIAFKDFNIFDGMAASPWVGLEHFKKL
FTSQEFGGVFVNTLIISLLKIVILFPLPIVVAILMNEMKNMLFKRTIQTVIYLPHFLSWVIVSGIFIDLLSTNGGLVNKV
LMSLGLEQIAFFMDSSVFRSVLITSAGWKETGWGIIVYLAAFSTIDPQLYEAAKMDGAGRLKQIWHITLPGILPTILLMF
ILRLGSVLEAGTEQILVMYNPTVYNVSDVIGTFVYRMGLGNQDYSFTTAVGLFESVIAFSLVITGNYLSRRYLQRGIW
>lcl|NZ_CP026520.1_prot_WP_042227925.1_2675 [locus_tag=PC41400_RS13620] [protein=carbohydrate ABC transporter permease] [protein_id=WP_042227925.1] [location=2951331..2952254] [gbkey=CDS]
MNTDAALPAVKPHKRRKPQTTWSEKLFHLINYTLFILFGLATLLPFLNLISKSLSDESAVISGQVSLFPVGFQLSTYAYV
LQDQMFLNSFKISITITVVGTLLSLFLTTITAYPLSKTRLRGRKWLLLVFIFTMLFSGGLIPMYLLMQNLNLVNTLPALF
LPAMVNVYNMLIIKNYFEGLPESLEESAKMDGASNLRILFSIMLPLSLPVLATIALFFAVYFWNDYFAAMIYITNPSLKP
MQLFLKELLISSTGDFLRNNNLDSSMNTTPQSIQAASILLATLPILLVYPFLQKYFVKGVLVGSVKG
>lcl|NZ_CP026520.1_prot_WP_042227924.1_2676 [locus_tag=PC41400_RS13625] [protein=extracellular solute-binding protein] [protein_id=WP_042227924.1] [location=2952351..2953844] [gbkey=CDS]
MLKKAALLLTVAALGVTTAACGGGKETQTGGTSPAPAGGAKPAMRQLMQYGNFDPNKEFTAKTISEKTGYTVNYEMLPAE
NPDEKLNLLMANKEEYDVMKLSAPQFEKLAASGALEPLDDLLEKHAPNIKAAIDPAAWESAKVNGKIYAIPETGSGLAVG
EELVVRQDWMDELGLKTPTNTDELYTVLKTIKEKKNIAPLTGSKDSKESIIGDIGAAFGVTADWVDQNGTLVHQVETPAM
KEFLTYMNKLYKEGLIDQEMPINTSAKAIEKFSSGKAAIFKQAYWNAGTTEKALKKNDPNAKTAIIPFLKDKSGKASTFV
KANTTWFVTIPKASKHKEDAMKFLDLKLKPDNFIEIAIGKQGEHHEIKDGKYYPILPKFNDELNNGSGFLTGVDYKKYPD
YWQARVRKDPVLQAFFEQINKNAEGIIVTDPLAKAPPIADISKSKQKLDKFQTDSMLKFISGADPLSNYDQFLAKWKADG
GDAMIKAANEWFKSAKK
>lcl|NZ_CP026520.1_prot_WP_042227923.1_2677 [locus_tag=PC41400_RS13630] [protein=S-layer homology domain-containing protein] [protein_id=WP_042227923.1] [location=2953956..2958845] [gbkey=CDS]
MKSLLRKWNGMMIIALVISLLTPAWGKASAQSFGQENGGVRPQAAAGVTDDVYVLSQDGTFRLPVGHINVNVDAPPQKTN
YVALFTSGAQVTNSPETDKVFVKKTNTAIAVDKHDQVIRVIGPTAVPPTGSTWEENQNLPVPEGGYVLLANDSSWGTSVT
RKPLFEHYKTGDTVSLHKGGQVVHAGDFLNPDPGLNLVTASGTTVTSPDFTVSGQVIRYGSGQGISLTVNGTEAALKADG
AFQSVVRLTAGTNAIPVKLLKDGREIVTSTVTVTYNDAQQPADLIEVEAAPIDITISIEGPRHAIGYVDQDIAGIDDTVA
LFTNDWGPQITVPQFNVAVQVDAGSKVTKVINPSIDGKTPAWTGPTDLEIPPGGYVLVAQDTSYASKNIKKYLATYFKVG
DAIKLRKNGFAVPVKDLMGTGGPIARLTLDNYAMYTETKPSTELSGTITNMDDPSKIALTVNGTPIPFSPDGKFKTSYTL
AEGINYLDLAVTKEGKEQDSKDLVVYSRPGFSTEKKVILWVDQAANARKFQTSDNVGNFLRTAKENGVTSVVFDVKGVEG
YVSYKKSTLTGRPYVSEIKAPEKAGSNPDLDLLQEFIRYSRELGLDIHVAFNIFAEGSIASNEYALLNSHLDWEERVYNA
ADKGQIKRLRESAKQGAVAFVNPSNDEVRDFQLKTIEEVLQNYDVDGVVLDRARYDNESADFSDLTKAKFESFLGARGKQ
LQNWPEDVFTYAGNERKDGPLIGDWWEFRSKTIKGFTSEVRQLADRYKAEKGKKIDVSAYVGSWFESYYLNGVHWGSTEF
RYDERLRMKDESVYTPGYYESGYVKNLDFIMIGAYQTTAPEIEHYITLGNIVTNGEVPLYAGIALTNVQEPALQREVFQA
GLENTHGLMLFDASQVNWPVAGAALRNLVYVRDYQLGISLPDSSDSFLEGSYYNTNLIENNIGVLTDTFGYSTGNSRFGV
EAVVDSSGKVTAVPNKTQAMTWNWGKPEETNSVIPKGGFVVSTLDASGIRTKRQLVANAYKIGDSVRSAALSGFLPYEGL
RTSAETVTLRGKVDVLGPGKAAVTVNGQEAALREDGTFQADTAIRPGANPVVITVRVDGFKTNEKTVTIIGDETAAKSLK
LDRGTYSMNKGESLQLAVTAEYLNSGKDVTGQAAYVSLDPSVVSVNASGRLTALKEGSGTVQATYEGHTATARVSVTSGS
TGGGSDTDSGTGSGSGGGSAGGGGTAPSGPERTSVTETKGSDGQNTTIVSADASVMEAEIAALQGKADPVLSYEIPGREP
VGIVSLPGTALRKAAAGSPDAILSVRSHLGAIELPAGLLLADLPADGSFDLLVQIARTAAGETAEIAARAAKEGMQVQGM
PVGFRVSLKSGGETKEIGGFGSYVRRTVNLPGAVDPDRATVVKLDPATGMFRFVPSRFETSGGAQRSVIRDRSGGTYAVV
TSDVSFSDLASHWSRKDVELLAAKRIVDGMEGGAFLPDEALTRAQFAALLTRALALDAAPAAADFTDIPGDAWYAGAVGA
AVQAGLADGFETGEFRPVEVLTREQMSVLLMRAMKLAGVRTADSAAGVLADFADRGEISGWAAGAVSQAVDAGLLNGRGD
GRFVPGGTATRAEGAAVLKRLLQTAGFVN
>lcl|NZ_CP026520.1_prot_WP_042227921.1_2678 [gene=aph(3')-IVa] [locus_tag=PC41400_RS13635] [protein=aminoglycoside O-phosphotransferase APH(3')-IVa] [protein_id=WP_042227921.1] [location=2959436..2960224] [gbkey=CDS]
MNESTRNWPEELLELLGQTELTVNKIGYSGDHVYHVKEYKGAPAFLKIAPSIWWRTLQPEIEALAWLDGKLPVPKILYTA
EHGGMDYLLMEALPGKDGSHETIQAKPEFFVKLYAEGLRNIHGLDIRECPLSNGLEKKLRDAKRIVDESLVDPTDIKEEY
DCTPEELYQQLIAKKPATEDPVFAHGDYCAPNLIIAGEKLTGFIDLGRAGVADRYQDISLAIRSLRHDYGDDRYKALFLE
LYGLDGLDEDKVKYYIWLDEFF
>lcl|NZ_CP026520.1_prot_WP_042227920.1_2679 [locus_tag=PC41400_RS13640] [protein=hypothetical protein] [protein_id=WP_042227920.1] [location=2960250..2960765] [gbkey=CDS]
MLYHFSEEPDIPAFLPRKPQGCEDLPPVVWAVDEGHAHLYYFPRECPRIVYTLTGDVGEEDKRLLFGPAASDTVIAIESA
WLERLRQTVLYRYTFEERDFRLLDATAGYYTAEEPVTPVSVEPVGDLLDKLAATGAELRIVPNLNPLRDALLASSVSLFS
VIRFRNAQPPE
>lcl|NZ_CP026520.1_prot_WP_042227918.1_2680 [locus_tag=PC41400_RS13645] [protein=Ger(x)C family spore germination protein] [protein_id=WP_042227918.1] [location=complement(2960778..2961914)] [gbkey=CDS]
MSIVKRSAWLGILLLLTGCWNSKDLQTMAYATALGIDYADGKYITYVQILNFSNIAKSENNQVGTRIPIWVGRGEGKTVS
ESLTSIYSTSQLRLFWGHVKAVIVSEALMKKGLPDVYDMLNRYREIRYNILFYGTNKPLDKMLSLTSMLNFSPIESLLTR
PEQYFSQKSLILPVYGYKFIANLNEAGNSTMLPSLTANSVVWEEDRSKVSLYEINGAYFFNGTKYTAWLSQEDLKGARWL
QTKLERTPLSVPKEHPIVALILESPKYKVDAHVVNEKIRFTISMEIKAFIDEFRERATEAEIETAAETDIKNEILMSFHK
GIKKRVDVLKLSETLYRKYPDVWKTHFKGKEFPLDKDSIEKIDVKVNISHTGKYKERK
>lcl|NZ_CP026520.1_prot_WP_042227916.1_2681 [locus_tag=PC41400_RS13650] [protein=spore germination protein] [protein_id=WP_042227916.1] [location=complement(2961911..2963395)] [gbkey=CDS]
MVPSSGNAEIWTAGQLEHLFDQSEDVVLLPHTFGENGPNSAEVLLLYGEATSNSQLIGELVLPGLEMAFDRDQGFPTAGT
VLSSCLPLLPFAGPITADTISESLFEGKLLLVFPHLRSAFLMNISSLPERKPEESNTELSIKGPRDGFVENLSVSIALIR
KRIRSHTLCCEKFVIGRRTKTKVALMYVRDILSPEVLLEVRSRLESIDVDGIYSISQIEEMVTDQKFSLMPLMDYTGRPD
FAVSALLNGRFVLIVDGNPMVLIAPSTMAMQIKSPEDLHNNFYYTSFSRLVRILSIFLCVLLPGFWVSLAAFHQDQVPFR
LLATITIARLGLPFSGQFELLILLILLEVFREAGIRLPSAIGQVLTVVGGLIIGDAAIRAGLVSPSVVVVGAITAVSNAT
LVNQNLSSAVSLLRFFLFFLSCLLGMFGLIMGIIVVTVYLSKLRSFGVPYLSPFSPPIYSNWVPAFMRLPWSLMTGRPKN
LHTQDKNHKKEDNP
>lcl|NZ_CP026520.1_prot_WP_042227915.1_2682 [locus_tag=PC41400_RS13655] [protein=endospore germination permease] [protein_id=WP_042227915.1] [location=complement(2963395..2964489)] [gbkey=CDS]
MAQSGGKISFLQLAMLLLLVNGLMNHVILNPIVLDASGRDAWLVPIFSALFLLPWCLLLVYFMKKSGQQKLQPWLAQRTH
PVVSWILIVPLYIQIYLIGMTTLLHTSIWTVTNYLPATPHFALSLTLIAVCAYCAISGLRTIAISAGVLLPFVIVLGYFV
MIANMPNKEYDRLFPMLEFGMNPVMNGMIYAGSGFMEFVLILTLQQHLKTRIKSWQLLVLALLFIHIMLGPLVGAITEFG
PEEAAKQAESPYEQWRLVKVGDSIEHVDFFSVFQWLAGTMVRISLSLFLLGDLLPFKRAKYRNRFIACMALSYLLITLLP
ITQQNLYEFMHRYYFHLSLIVILGSTLVWTLISVTTKTGKEGTS
>lcl|NZ_CP026520.1_prot_WP_042227914.1_2683 [locus_tag=PC41400_RS13665] [protein=antibiotic biosynthesis monooxygenase] [protein_id=WP_042227914.1] [location=2964873..2965199] [gbkey=CDS]
MFIQTRSVVVEKGNAEQVVERFSKEGAIDSREGLIDVSVMVNRSKKENEEVLILIRWESEEAWKNWEKSEEHIQGHRQNR
GQPKPEYIISTSVNSYELKKRKIGKAGQ
>lcl|NZ_CP026520.1_prot_WP_042227913.1_2684 [locus_tag=PC41400_RS13670] [protein=DUF1697 domain-containing protein] [protein_id=WP_042227913.1] [location=complement(2965421..2965963)] [gbkey=CDS]
MTTYLALLRGINVSGQKLIKMAELKNMFEAMGFLAVKTYIQSGNVLFRSDLANSEEALRGRIEDEIAAVFGFQVSVIIRT
AEEWEQIIRNCPYDAADLKEGESIYVTLLADRPAPEDLGRVPEPDPRLEEYQTAGRDVYLLFHQSIRDSKLAASLHKLKT
PATTRNWNTMNKLLALSRSL
>lcl|NZ_CP026520.1_prot_WP_042227911.1_2685 [locus_tag=PC41400_RS13675] [protein=helix-turn-helix domain-containing protein] [protein_id=WP_042227911.1] [location=complement(2966302..2968830)] [gbkey=CDS]
MRLKRVLLNRHSLVYKLFFSFLGILLLFSVFNGVSVYFFNKSMQNEIIQYNRLMLKNTAERYETHFERIKTMLFDLYSNE
TVVAFNRQLLTKAEPDIDYWKASEVLKTVRAQAYNPMFYLNNLLVYYDSHSLIVEKEGSVSAPMLFERFYRSADYPLVYW
KEQFARSGNYVLHPQSSFNVTTLNSSVDVGLIPFSFRMPGSNYQIVALLDARALQEAFYGPGDSREFMILREDASLLYRS
SSAFGAGDIPSFGPEGAGYAKAGDTYFFAEKDPLTRLTYVTALPYANITAKVRNASLTLLLIFAVALLLGIAASLFFSRR
IHRPVKQMLSSIVSRDPVRPKSAIREFDLIHQNLHELMQEKEAVQKELLGKKSLLTSFGYINKLKAITSNINDWQDIAEM
DDPFRLVLAQLHFKEGTVPESHSRPDRIAYYIREYINVVMTEHLPHSHTFQIDTNQILTLVKGGDEPEKLGEALEKLKPI
LDRDKDYFLVTLAVGSVHQGSSQFNAAYRQAQEMAQQARPLDECQIIRSYRVSPPYAPTPAQEQELFRHIQAGNLAHCVS
LTERVVEQMEKKDAGLAQYRQIAESILLRAARAVEPYGRAAAAAVARPDSGADPRQASPATDASAHRPAETPPDGGAAER
AAAAGKGEGTGGAGPAVSGTATTCGGDPGTPCEAACACGGACGPEEALSFQERHRRRLRTCYTPDQFRRFYAQMFKEAAG
IIAAGKEERDSVIEFVMNELENKFANDISLDLLADRLNLSAAYLSVYIKEKTGANFSEQLNAIRIRRAKELLGGTGLSVQ
EISVRIGYRNVTSFNRMFKKMTGLPPGEYRKSQVIGGEFGGL
>lcl|NZ_CP026520.1_prot_WP_042227909.1_2686 [locus_tag=PC41400_RS13680] [protein=ABC transporter permease subunit] [protein_id=WP_042227909.1] [location=2969021..2970004] [gbkey=CDS]
MSQEASSSQTAAPASAPAASAVRKPKRSAFFKRVRRNWDLYLLIMPVLVFFLVFEYFPMYGVQIAFKNFIATKGIWGSPW
VGLKHFERFFESYYFWRLLTNTLGIGLYQLVVGFPVPIILALMINEVRSKKFSKFVQTVTYAPHFLSTVVLVGMVFIFLS
PQTGLLNMAVTWFRGEPVNFLTDPAWFKSVYVLSGVWQQMGWSSIIYLAALTGIDPQLHEAARVDGASRWQRIRHINLPG
IMPTVTILLILNMGSLLGVGFEKVYLMQNSLNMQASDVISTHVYQKGIIDGQYSYSAAVGLFNSVINFILLLTVNRIARK
VSSTSLW
>lcl|NZ_CP026520.1_prot_WP_042227907.1_2687 [locus_tag=PC41400_RS13685] [protein=carbohydrate ABC transporter permease] [protein_id=WP_042227907.1] [location=2970021..2970908] [gbkey=CDS]
MNTIQSTREKAFDWAVYALLTIVSLIVLYPLVFVLVASFSSPEAVMRGEVWLWPKDITWMGYTKIFQNNEILTGYLNTIV
YTVVGTAINLILSVAAAYPLSRKDFAGRNAISALMVFTMFFSGGMVPTYLLVKNLSMLNTMWALVIPGAVSVYNIIIMRT
FFQSSIPGEMQEAASIDGCSNLGILWKIVLPLSMPIIAVMILFYSVGHWNSYFSGLMYLTDRDKYPLQLFLREILIQGQM
QEMLGVGDDSHARNVMEGEAIKYAVVIVANLPVLMLYPFLQRYFVKGVMIGSVKG
>lcl|NZ_CP026520.1_prot_WP_084706552.1_2688 [locus_tag=PC41400_RS13690] [protein=extracellular solute-binding protein] [protein_id=WP_084706552.1] [location=2970999..2972630] [gbkey=CDS]
MRKMRWTAVTLTVAMAVSVLSACSGGNEGAGSADGKDSKKEVANLNPTGMPIAKEPVSLTFFTGKAATNGNNFEETLVWK
EYAKMTNMNIKFQLVPFENLTEKRNLALAGGDYPDAFYSARVPSSDLMKYGAQGTFVKLNDLIDQYAPNFKKLMEKYPDL
KKGLTMPDGNIYSFPSFYSPEFLPMLIGTPLWVKQDWLDKLGMKEPATTEEFYQYLKKIKETDLNGNGQADEIPYAGEGI
NPLIEQLRGAWGFGNRGLGHKFVDVDPKGGDLRFFRTDPGYKEVIEYMRKLYTEGLIDKEIFTIKTAALFAKGQKGVFGA
TINPNPKTQMNQTGYFGLGALKGPHGDQLYSHVKVPAVWPGAFVITDKNKNPEATVRWIDHFYGDEGATFYFMGIEGQTY
TKTADGKLEFVEEITKNPNGLTMDQALAKYITWLGGSYPGYVQQKYFKGSETLPESVETGKKAEPYAVKELWNNFNFTEE
ETEFRSTVGKDMETYIGEMEAKFITGSQPMSEWDKYVATVQKMGVDEYMKIYKQSYERYKSSK
>lcl|NZ_CP026520.1_prot_WP_241688862.1_2689 [locus_tag=PC41400_RS30285] [protein=AraC family transcriptional regulator] [protein_id=WP_241688862.1] [location=2972881..2972985] [gbkey=CDS]
MKIFDVARLSGYQSPKHFMLVCKQKTGQTPGEYW
>lcl|NZ_CP026520.1_prot_WP_084706551.1_2690 [locus_tag=PC41400_RS13700] [protein=helix-turn-helix transcriptional regulator] [protein_id=WP_084706551.1] [location=2973068..2973205] [gbkey=CDS]
MSDSGYRIKCIRKENNLNQSQFAKSMGISQGNLSEIEMGNSNLRR
>lcl|NZ_CP026520.1_prot_WP_084706550.1_2691 [locus_tag=PC41400_RS13705] [protein=transposase] [protein_id=WP_084706550.1] [location=2973533..2973847] [gbkey=CDS]
MTCQFGTKQCRSRWRHRVVMFDDGHGREIRAVTNLLGTSPEQIADMYKAHWKIELFFRWIHQHLNVPTLFGTSENAVYGQ
LYVALVVKLAFLNKRERLITIDES
>lcl|NZ_CP026520.1_prot_WP_042227902.1_2692 [gene=loaP] [locus_tag=PC41400_RS13710] [protein=antiterminator LoaP] [protein_id=WP_042227902.1] [location=2974047..2974595] [gbkey=CDS]
MKYWYALFVESNQEEIVLQHLNLHFDCSVLNAVVPKRRVPERKNGEIQHVIKKLFPGYVLINCRLTDEIYYKIKRIPRCY
RIVHNGKSFNQEECKIEPQEIEPIIKLLGDRDTLEYSSIYLVKSKVLVTSGPLKGLEGIIKKVDSRKNRAKIALNFLGNE
KLVDVGVEIIHKLNPRSYEPSN
>lcl|NZ_CP026520.1_prot_WP_241688863.1_2693 [locus_tag=PC41400_RS13715] [protein=ABC transporter ATP-binding protein] [protein_id=WP_241688863.1] [location=2975742..2976674] [gbkey=CDS]
MIIEVSDVVKKYNNLDVLKGVTLSIREGEIFGLLGPNGAGKSTLMNTILGLLRFNSGSVKILGKDFSKHATEIKRKIGYV
PQEIAVFETLNAIDNVTYWGRLYGLRGNELAQMVKKALEFTGLWDRRKSKVSTFSGGMKRRLNIACAIVHKPRVLFMDEP
TVGVDPQSRNHILESIRIMNKEGTTVIYTTHYMEEIEAICDRVAIMDFGQLIALGTIDELIADHIHEVCLRLEFTANAAE
SVKVIESFDGVLSFSVNEEILEIKLDKDDSRIPLILEQLIKQHFNIKYMAVEKPNLETVFLQLTGKKLRD
>lcl|NZ_CP026520.1_prot_WP_042227898.1_2694 [locus_tag=PC41400_RS13720] [protein=ABC transporter permease] [protein_id=WP_042227898.1] [location=2976695..2977843] [gbkey=CDS]
MNILFQMFYAVKRMIMNPKAILINLLTYAIVILILGSVFGSTYDIQKSLGKVTIAYANLDSGRNGAIVDAVLHSNDVTQL
AELKPVMSLDDGKVLINKDEADVIIYIPDNFSDDKDLPHVVKVYLSTDYGVPTMVVKNIVSSLTNMMNTSSVIYKITGQT
EVDAMASELQKNNEENVKEQSLTSSSLQATTAMGYYSIAMILMLLLYGQEYGIFMASEDYLGPLGNRMRLSAVKPYVQYI
GKGLGLSLVTFLQGVILLLFTKYVYGVDWGNHIMVVLLTLFILSILVTLLGLMLVIITRSIQKAQSISNFISLGGTFIVG
GFVIADFGSVAYLSPSYYAKTALFNVVYGNQVEQALINIGVMILICIGFVLVSLVASRRTVE
>lcl|NZ_CP026520.1_prot_WP_084706549.1_2695 [locus_tag=PC41400_RS13725] [protein=ABC transporter permease] [protein_id=WP_084706549.1] [location=2977840..2978970] [gbkey=CDS]
MTIFLNCIKRVFQKGWSLLFIFVIPIILIVFLIVVKSQEVQYKIGIIDEDNTLFTQDFIEQLEGQYKIINLTSNDDIKSK
VINSELDGGLLFQQGFTESLLDGEDVSVLIYDIAGTDIFGPVKVYVSSYVSSAKKIASASGGDEKTFYQGMDYYKNSAFN
VTYESSLNYAGSERDVSNAVKSLGFIASGMIFLMTFSTSLILQDKLSGVYDRIAVTPISRFSYLLQNMLACFVIAAIQSV
LLLSIIASIVDFPFGQTFEQKQEVLLVCLAFSLVCVALGIAISRFSNTRLMAGSLSTLIYFPMLMLGGCFWPREIMPKFA
QQIGDFFPTTWFLQAGKDVIAGKGIAAASQQLIYLLCFAALLIFVSFIVRNEKVRT
>lcl|NZ_CP026520.1_prot_WP_042227896.1_2696 [locus_tag=PC41400_RS13730] [protein=ACP S-malonyltransferase] [protein_id=WP_042227896.1] [location=2979084..2980151] [gbkey=CDS]
MSNIAILLPGIGSQYLKMGKSMYDNYASAKRIFDIADECLGYHLRERIMYGDSKEFNDKTFVPAAILTTSIAFFTVFQDV
FNIKEAYYAGHDIGELSALVCSGAIDFSDALNFLVEHSQITLSQNDASNKEEISFRLQELISKLTLKRHVENVITTVDGI
IYTDSEDIVRLMLQPYTQTRWKDTIRYMTNQGIYRYIEIGPSNMLRPQVLDVEPAARVLTVDYEGDPYYSTELFKKRKLF
NKTYLLSRMLGVAVSVPNNNFDDRDYLQNVVTPYNQLKDIYDKYSSTNEQPSDSDMADASRYFMQIMHGKKTEANEINLR
LSDLQEETLIEEVDMFIDLNKTTLLAKGTPVSGMS
>lcl|NZ_CP026520.1_prot_WP_042227894.1_2697 [locus_tag=PC41400_RS13735] [protein=polyketide synthase] [protein_id=WP_042227894.1] [location=2980167..2982065] [gbkey=CDS]
MVTTKKTNAMDLVISMFKDGRIDKAEAGQLLKGLNSQKTPVAIVGMGWKSPLSDNYNDIWDVISNKRTLVQACPKDRITR
AAQLLGGLDTDERNYHYGAILSDIDEFDYELFGISHEDASLMEPSHRIMLQAAYRALEDAGYLGTDRRNEITGVYIAANF
TANQLVNYLKHIDDINMKSYMLNWTSFLATRLSNKFDLRGPSTVIENSCVSGAVALYEACNLLEMGKITSALVGGINISH
LLDQRIVLNEVFHHAPDIASKGFDDEPGGNYVGEGAIALYLKPLDKAVADQDRIHGIIRGIDTNNSGGATIDFLQTSSEM
ITQVVQGALIDANLSAEDITYVDGEGYCERIEQALEVKGLINGFQTSTQKKQYCALGLTSVNLGYSEAMLGFSNAVCSVL
ALKNKQIPPVYKFDTPSSYLDLINSPFYINDKLKDWEVEEGKKRIVASFSSGYGGGNALTIYEEYIPDERNLQPWIKDCY
LFCISGHTLVSLQQTISKFTEFVKQYKDTETANALDFSYTVLARRQHYSPYRLAVVYRTFDDLYDTLSQWIVQNRSSHDI
YYDEKRRKSVAENRNVEQGMLSSFARYDMREIAEYYVNGYAIDFNILFHDEQPRLIDVPGYCFTQSKCWIGG
>lcl|NZ_CP026520.1_prot_WP_042227892.1_2698 [gene=fabD] [locus_tag=PC41400_RS13740] [protein=ACP S-malonyltransferase] [protein_id=WP_042227892.1] [location=2982100..2983344] [gbkey=CDS]
MMRKAFYFPGQGSQYVGMGKTLCENSKIASDVFAEASDALSFDLKKLCFESDQANLTLTQNAQPAILTTSVAMFRVLMDR
HMIKPDLMAGHSLGEITALTCAGAIDFADAVRIVRKRGELMQEAVAPEAGCMVSVLTRDVEKLEHICHSVTQSNELVSIS
NYNSRTQTVISGHRKAVDKVVTLLNEEGLKSVYLNVSAPFHCGIMQPVATLFEEELNKYRFTNFTIPVLSNVTAKPYLGS
EDIIPNLTAQIYTPVRWVDCMLYAKKYMIQYGVEVGPGHVLKNLMNNIFGDTPIFAYDHTNDIEKLEKHIQNTAIPFLSR
SLGIFAATRNNNWDSEQYQRGVIEPYSKLSALQSEIENEGRIATEDEMQQAITMLLTMFKTKQTSREEQIARLKELFRDS
NTETIFEHFDYNAI
>lcl|NZ_CP026520.1_prot_WP_042227891.1_2699 [locus_tag=PC41400_RS13745] [protein=SDR family NAD(P)-dependent oxidoreductase] [protein_id=WP_042227891.1] [location=2983365..2988035] [gbkey=CDS]
MDLNMIKQIPYSLSDSDCHEPSAQDIAIIGLHAKVGKANNVNEFWEYIRDGMDLITDFPQQRYQDAVRSYQCMQQKEMPE
GLGQCAYMERIDLFDPSVFNISPREAEFMDPAQRMFLESAWTALEDAGYGGGCLKGSPTGVFLGYTSPPDSYRSLLKESD
AHTYGISVSGNVDAIIASRISYLLNLKGPAVNIDTTCSSSLVAVHMACEQIINGSITMAIAGSVRFRLTPAPQVIENKMG
IESSSSRTKTFNYDADGTGTGEGVICFILKSLRDAVRDRDNIYAVIKGGSINQDGASIGITAPNAEAQDSVIRSAWKDAR
INPETISYIEAHGTATKLGDPVEINGIERAFSRVTDKKQFCAIGSVKSNVGHLDSAAGAVGLLKAVLMLKHKQLPPTLHF
TAPNQKINFSSSPLYVNDKLRYWEPTHYPRRCGVSSFGMSGTNCHLVLEEAPQVESVIDSPKQSRIFTVSAKDEQTVRRY
IELYKQHLTQMPNCNLDDLCYTANVGRVHFECRFAMILHSPDDLFAISFTQHANDNRYVFGEHKITDSDITDGYIHRDTV
QLLSQKAADITQQIIHEPDEGHHKELLNELIALYVQGAAIDWEQLYGKEERRKLSIPTYPFDHKRYWVSLPSRQEKELFE
SRKENLHPLVHTCVLDTHRMHVYMTKMNVEQCWELKCHMINGVHVLPGTVFFEMAVFVSTRYLKHAQFDIENLLYCVPFA
CGRDETRTIHTIIREEHDVITVNCFSKNDKEVEWIHHLEVSVRTRKHASPKNRSIDITSIIARCEVMDKHTSLAPKSIVE
ITGNKWNNLSALYKNDKELLLCFSVDKTLLAEVDQYILFPSLLDPAINSGNLIINDVFLPFSCEQARFYQPLPPRMYSYI
KLKEQERVTQEFAAFDVTLLDEDFHVIGELNNYVIKKIHDEENFMKSSQQQYRTTHWIACSDPIDKTVNVVKEDDCILIM
YRRGQLNSSLMLEIRQYFGKRIIAVELNETCDFDTILSSLAKENIKYIVHIASLIHEEMQSVSQVITEMELLFESTFNLI
KALLNQDVRHSINLVIFTSNATKVTGEEERIQPLSYGLSGMVNTIGREYANIKTRVIDIDNDTSADVLIEDMLSNELLHA
VSYRKNVKYMQQLNNVTHGTTLDDEQMTLVDNGVYIITGGLGGMGLAISQYLFRLNPTIHIVLLNRTYSRDTFSLLSEPL
DPTLRKKLEQVQSLWSQGYSLDIMQVDIADYVQMDDALQQLRDAHGAIRGVIHAAGIAGDGFIMNKTWHTYESVLRPKIA
GTWVLHELTRDDPLSFFVMCSSYASIFGAAGQSDYVAANAFLDSFSYYRRTLGLPSLTINWTGWRESGMAVVSGIREEGV
YVRFLNDEEGAAAFWHALKTKLPQVLIGDIDYKALEIEHDNLIISLFENGSNHRGNDQSHWSKGTSLEDKEIVVIGKSMS
DITDIERNVIYAWSQTLGKDQVDIYAKFFESGGNSLLAAYLQKELDKFYPGIIVITDMFVYSTVVEIAQYIEAKTMKSDK
ITVKKESSNNIQDMVDQFVSGDMDMEQVLALLDKDN
>lcl|NZ_CP026520.1_prot_WP_042227889.1_2700 [locus_tag=PC41400_RS13750] [protein=SDR family NAD(P)-dependent oxidoreductase] [protein_id=WP_042227889.1] [location=2988052..2992947] [gbkey=CDS]
MNTVKRFILDQLSMKHISQTEAVNLLAKLQQNEEEKLTEIAVIGVANRLPMAKNPGEFWGNLVNSRVCFVPKPEDRLVFD
KMLLKNSAFAEYIGYTPSPENYDKLEQFQGGYVTDWDKFDAAFFGIPPREARYIEPEQRVFMETAVTAIEDAGYSIPSIT
NSNMGVFVGKDETNSIHYKYLTEDDQMKLSGFWNGILASRISYTFNLKGPALVLDTACSSGLVAIHEACSALRNQECDMA
LAGGINLGITGFNMQDSDNKETSDEDMSIMDTVQSKDYRIRTFDKKCSGTVFSEGVVVFLLKPLKKAIKDGDHIYGVIKG
SSVNNDGASNGITAPNPLAQEEVIINAWKRANVSPESISYVETHGTGTLLGDPIEVTGLSNAFSKFTDKKQFCGIGSVKT
NIGHTVGAAGPTNLLKVLLSMKHRMLPPSLHFEETNPYINFINSPLYVVDKCMDWDSGEMPLRAGISAFGFSGTNCHIIV
EDYKDSDLLPDTTSNNVLTLSAKTETALINLIHNYEQFLENEQDIPVGNLCYTANTGRGHYTYRIAIVFNTLKDLTDKLA
ILKQNDLKNMVNHSIYYGKNHVVSDKRQSINEGDISTSELSRITQEAQKLVLTGREVSENISQLSKLCELYVKGATVDWK
QLYQNGSYKRVSLPTYPFDRTHYWAEPITMAQSITRKPDNTILHPLVHQCIVDSMKQSVYLINPELTKEWVVTDHKLMGT
HLLVGTAYLEMIKEACQHCFNNANIVIQDLVFMAPLLAYEGDMDIHIVITQNDVNADFSVVSRHEESTGDVWIEHSKGTV
SMHDDTPPKQLQFSEIESDPALKKFSIKQGLEEKELILNSWESDNYFGERWESIMNFFTKANGDRDEIITEIKIPDKYLS
DLDHFSFHPALVDNAINIPVIRLYTGGRKLLPFSYKNLKFYRRVPNHFYSKITILTKDNNTEVFSFRVSLVDISGNVIAE
AEEVTLRQISKMNEFKKNAFYGVNWMLQPTTIQKTQNVQGNILVFCDQGEYYARALAERIQHPDSRLFFVSIGNKFSQID
ATQFTVGEDEEDYDKLLAAIGVNNLCSVYHFSTVNFTRTDVSAEQYGEELHQGLYSVVYLTRAILRTIRDSVEFVLMTDY
AHEVTGEEACIKPAHAAFLAVTKSTATEYPNLRYRCIDIDEATDMETVLDEVLTENPTFRVAYRNNQRFTEALTVIEQKP
ESIRELEIKPTGVYLITGGTGGLGLQMAIDLGNRGARTICLLARKALPDRSTWQTLLSNGDRKLKTLIESILYLENLGCT
ILLRYTDVGDRVQMESILSELKTAYGPINGIIHCAGMAGDGFLFNKNMNTFHDVIRPKINGTVVLDALTREETLDFFILF
SSMTALLGGPGQSDYTAANAFLDAYAPYLRKKGIPAQTVNWPGWSEVGMAVDYQVADAVTLFRSLETEVAIQALNEIIKY
DLSNVLPGQINYDILQQIGENNLPFRLSAWLKKNLDRHMSRHNTQRKQVAKQTITPEDIHVIGKNDNYTTTENEVAFRYA
VVLDLREIDIYESFNSLGGDSIHAMELLKELNAIYPRTIDITDIFTHSSVEELAGLIDKKRGIGKEQSDDKAFIVPDYKL
SDDDIMSMLDNVEKGNVSSLDEAIKVIKSNK
>lcl|NZ_CP026520.1_prot_WP_053228668.1_2701 [locus_tag=PC41400_RS13755] [protein=thioesterase domain-containing protein] [protein_id=WP_053228668.1] [location=2993003..2993740] [gbkey=CDS]
MGNEIKLFCIPYSGGSAEVYRKWSKSLYSEIQLFPVELPGRGKRIFEPLKDDIDSIVEDITATIASNVTSNDDYAIYGHS
LGSLLTFETYYKLIENGVHKPQHIFFSGRNAPQNMHNQTSIHLLPDEQFLHAVMSYGGSTHEIIQNPELLDLFLPILRAD
FKISETYCYQEKKDKIACDITIINGRNDHSAIRYDMAEWKYYAGGAANFHTVDGGHFFITENYGPVVDIIKQTLTISDKI
LERQG
>lcl|NZ_CP026520.1_prot_WP_042227887.1_2702 [locus_tag=PC41400_RS13760] [protein=hypothetical protein] [protein_id=WP_042227887.1] [location=2993799..2994023] [gbkey=CDS]
MNSKIIWGIVLLALSLFLKPIMAIAGLNERLFGLHLQGETLFSLSPFLVRIILAVIGVIMLIIGGMQTANQKKK
>lcl|NZ_CP026520.1_prot_WP_042227885.1_2703 [locus_tag=PC41400_RS13765] [protein=non-ribosomal peptide synthetase] [protein_id=WP_042227885.1] [location=2994045..3014585] [gbkey=CDS]
MRKFVPNKKHEEDRSSYWTYLLDDDSSTAIKPLFPYRPSSKRTVGTNQMDWVFPQHVCEKLRKLSREHNIPMNVIIEAAW
GLVLQKYNYTDDVMFGKIIDDPHLNSVMIPVRMKCNRDDVLFDLLQQLKEQEAESRKYAHCDILPRKISHLIKTVCIIKD
RTDNGKQLIHLPLEDQLSHNPPDLLLHFSFDCNQLNVMIRFDESKYTKSEINNIRTMLTFALCAFAENSESKVRDISFVS
DEDRDKILTLFQTKPTQYPHEESIKSLFEAQVKKTPDRIAISCGTEQLTYAELNAKANIVANRLLAAKVNSGTGKGETMV
ALLLERNMDMVIGILGVVKAGYVYVPIDPEAPQDRIQYTLADSGASLLLVNQGKEQMVSLPQGSIYAIEQLYSDEPDAAR
GAELKANPDVAIAPQDALYIIYTSGTTGEPKGTVIEHECVVRLMHNDDMPFHFHEEDVWSLFAAYNFDVSVWEMYGALLY
GGHLVVIPNDAKKDSYALLDLLREKRVTVLNQVPSSFYNLMQVELAEAGERLYVKHLIFAGEALQVGKLKAWQAKYPEMN
IVNMYGITETTVHVTYKAITAEDIALDGNDVGKALPTLKIYILDGDKLCGIGVPGELCVAGVGVARGYLNKPELTNKKFV
ANPFVPGERMYRSGDLAQWLPDGNIAYLGRMDDQVKIRGFRIELGEVENALRQLEGIQDAAVIVRETHDGEKALSAYLIS
KDIVQLQAVRRDLEEVLPAYMVPAYMMQIEQLPVTRNGKLDKRALPEIVAVSEKEYTAPKNELEEQLCNIFSEVLGVERV
GTQDSFFELGGDSIKAIRIVSKLRSAGYCVASKDIMQKYTVEAISYAAQKSALEEQYEQAEVSGIVPLTPIVREFASWNL
PKPHHFNQDMLMEIDLEDEKQLREVLNALTAHHDILRSVYREGQLEIVKVSDSRAYELKVYDYSEEQAAAELMEAACTDL
HSSINLEEGPLVKAAMFRTGEGNLLFLGIHHLVVDGVSWRILQEDISTAIRQVKEGTKIQFPSKTASYKAWAEALAEYKD
SHLLLKERPYWEQVAAQMAAGELRMDEQTGESGYDSYTIHLSVEETESLLQRAGKAFRTEVPDLLISALGMSVKKLTGAS
VVTIGVEGHGREAIHRRIDIDRTIGWFTSVYPIMVACHDEMAESIIATKEMLRKIPNHGFGYGLLYDLLTGSQLPTLLFN
YMGQMDAEAKGKKINFLSSSKGLADENKMLSELFINGVISEGKLSFAFRYNRGLFSNDTIQRFAEVYKGCIQSTLTYCMQ
VEESVPTPSDYHAADLTRADLSVIQQAVGGSSQVERIYGLTSLQKGMLYHSIAEPESTAYRNQNIFIGQGYADEHMVKQA
LKLLVMRHEVLRTAIIHKNISIPKQVVIRGREVEYERTDLTGVEPSVQAKQVEAIAQDQIQRGFDLEHDPLLRVHHVVLA
RDRYQLIWSFHHILLDGWSLPIIYGDFNRIYKALEKGALTSTLETQVITEKQYQDSYEDYINWLEKQDQQKGLSYWSDVL
AGYEEVAEIKPIYTPQESEQQVQSMAITLSTEDALQLRDLIATHQLTVSNVVETAWGVVLQAYSGQKDVVFGKVMSGRQA
EVRGIEGIVGLFINTIPVRVASKESMSILELIQEMQQQGTESEQYAYCSLAEIQAQTVQKQHLIQVLYMFENYYVDEDKL
TDFMYEIEYGLDQTNYNLTLCAYEYEGKIVCEIIYNPNVYAREDIAQIVVRLEHVIHAIVANPEGKLSELETITEEEQEQ
ILGAFNDTNLPYAKDKTIPALWEEHVANTPDAIAVVFEDNHVTYDALNKQINQVAWALRSLHIKPDDRVVIVAERSVEMI
AGMYGIIKAGAAYVPVDPTFPAERIQYILADCKPKAVLVYKATIETDLPTINLADDMLWEGKSVNPKQINRPNDLAYIIY
TSGTTGQPKGVMLEHGGVVSLRSHLLKQYNVTSQDNVLQFANYVFDASVWEMTLSLLLGATLTLVSKPIIADVTRFNSFM
QESRITLAVLPPQYYLQTAFTGVRWLTTAGAATNESVIEQTRNNDCGYCNAYGPTECTVQATYWEYDGISDVPNNIPIGK
PIANNHVYILDGDKLCGIGVPGELCVAGDGIARGYLNKPELTSKKFVTNPFVPGERMYRTGDLAKWLPDGNIAYLGRIDE
QVKIRGFRIEPGEVENALRQLAGIQDAAVIVRETHDGEKALSAYLVSKDRVELQAVRQDLEEILPAYMVPAYMMQIDQLP
VTRNGKLDKRALPEIVAVSEREYTAPKNELEEQLCTILSEVLGVERVGMKDSFFELGGDSIKAIRIISKLRSMGYHIAIK
DIMQRNTVGAISYAVQKSVIEEQYEQDEVSGIVPLTPIVREFASWNLSNPHHFNQDMLMEIDLEVEKQLREVLDALVVHH
DILRSIYREGQLEIVKMSDSRAYELKVYDYSEEQEPAGLMEAACTDLHSSIDLEEGPLVKAAMFRTGEGNLLFLGIHHIV
VDGVSWRILQEDISTAIRQVKEGTKIRFPAKTASYKAWAEALTEYKSSPLLQKERAYWEQVATQMESGALSMDEQTGESG
YGNYAIHLNEEETTQLIHQAGRAYNTEINDLLISALGMTVKNLTGARNVTIGMEGHGREAIHKRIDIDRTVGWFTSVYPV
VVSCHDEIAKSIITTKEILRKLPNRGLGYGLLQEELSVRPLEIMFNYMGQMDAEAKGRKLHFFSSGKGLADENKMLSELF
INGVISERRLTFTFRYNKGLFSSSTIQRFAEIYKACTQSMIAHCMQQEETVRTPSDYSAISITDEDLSVLHQLYPEKNDI
QDIFDLLPLQEGILYHGLKSSGSEYLTQFIYQINQHIDNDLMRSAFQCLLKKHDALRTAIVYEGLKHPKQIILSHRSLDW
EEIDLTTYDEKEQLARLNELIEQDVKRGFDLQLNALFRIKNIKLSTNCTQLIWSFHHIIADGWCSELLCNDLKNYYNLLQ
NGYSTHDVEQAIDQEKFQVATYGDYIQWFQNQDEELALTYWRERLRDYDLTYKYRPYFDPPYSSSNLAQVKKQLDVELTD
SLLLLAQAHQITINTITEAAWGIVLQQINFTNDIVFGKVISGRHAEVRGIEKIVGLFINTIPVRVSADEDTTILALLQEL
QDQGIESTMHAHCSLVKIQEQTAQKSDLISSLFVFENHLAQEQQSEHTDGLQLALVSSRDETNYPLNLLAAVEDKALRFT
VLYNPNKHALEEVEMMLSMLNNVLCAFAEYPEKKIFDVELIRKEDKARILGEFNNTASTYPKHKSIPDLFEEQVQLTPDQ
ICVVCDKEQLSYQQLDKAAQGIATELRKRGVSSGQLIGIVCEKSVHMIVGLLGILKAGCGYVPINTDEPDNRLRFILEDC
DLNIVLCGDVKGDMINVLQEYCTVIDLNGSYAAMTERALVRADDMAYVMYTSGTTGYPKGVCVTHQNVVRLVKQTNYADL
NGAHILLTGALSFDASTFEIWGALLNGGKLVLAHMQNITDGNMLGKLIYEHQITMMWMTAQLFNYMVDTRCEVFQPLQYL
LVGGERLSAKHIRAVREKHPTLRVTNGYGPTENTTFSVTYAIETDYFNIPIGKPISNSRAYILDRKMQLCGIGVPGELVV
GGDGVAKGYMNQPKLTVEKFIENPLIPGERLYRTGDIAMWLPDGNIAYIGRIDDQVKIRGYRIEVGEVEHALQKLEGIQD
AIVIAREQDGSENELYAYLVSSTEIQMHNVRNSLGKTLPAYMVPQHMVQIEKVPMTRNGKVDKRTLSEIQIVHEQQYTAP
TTEQEFILCKIFGDILSVDRVGIYDDFFALGGHSLHAVRVINQIEAMIGIRLPIITLFENPHIKALSQHLELISEGNTSH
NTLYPKAVFEPIPHAMQKETYRMSSAQKRLYAIQEMDQHSTAYNMTLLRRMDADLDSERLQLALKQLIQRHESLRTSFHM
ENGEAIQRIHDHVPFTLLITEEKDDDVTGFVRAFDLSQAPLIRVKVINQKEDGYLLAFDMHHIISDGTSINNLMQDLSKL
YNGEPLEAMPIQYKDYSEWLQTQDMTGHETYWLNSLQGELPVLNLPTDFTRPQVQQYTGKAVVSHVELREAVTALCRMSG
ATEYMILLSTWMILLSKYSRQEDIIVGCPVNGRLHYDTEPVIGMFVNTIALRGKPSGKKNFVDFLSEMKEVCLKGIEHQA
FPLEDLVDLMGLHRDMSRNPLFDVLFVLQNNEEADFDFYGINSTNVHDEYSLQHSSAKFDLTLEVTPMASGYRIEVRYAT
SLFQEESVRRMVKHYETLLGAVTQSPKLEICQLPMMTQAERQHILTSFNDTSCVYDVTTPIHEQFEQIVQAMPAKTALRF
RDQALTYEQLNVRANQLARTLLLHGTRRGDFVGILFGRSIEMMVGILATLKVGAAYVPIDPTYPLERIQYMVEDSGLSTV
LTQRQYDDTIELPSHVRRIHVDQDGCDEPSHNLHTIVDATDIAYIIYTSGSTGKPKGVMVEHRSISNLIAGLNQLQNLDS
MKTMLCVSNISFDGTVYETLFALLKGMQIIIADEEAQHDPKLLLQLIEQYDVNIMKGTPTRMKMLMDEVRDVHPLHGLEV
IVVGGEPLTDDLATALLENWNIRLFNVYGPTEATVDATGIEIHLHDKVTIGKPIVNVQVFILDSYGFPLPIGVAGELCIS
GAGVARGYLNRSDLTAEKFVDNPFLQGERMYRTGDLARWLPDGNIEYVGRIDDQVKVRGYRIELGEIEHVLRKQPDVLNA
VVMVREDAVGDDYICAYVVPNDQVTLDTEAIREDIRQVLPDYMIPAYIVPLVQFPITNNGKLDRRSLPEPDRKGRGNKYV
APRDQTESVLANIFREVLGVERISIKDSFFELGGDSIKAIRIVSKMRSAGYLISVPDILSQYTVEKIAYTVGRESEIQYD
QGEVTGQVLNTPILHDFCARNYANPNYHNQDALLTVRTDNESHIHEALKAVAVHHDILRAIYRDGSLTILSSTDSKLYEL
QVFEVPADLNGSTWIENASAIIHRSMDINKGPMMKAALFKTESANYLFLCIHHLVIDAVSWQFLLEDLETALQQLKDGKE
IKLPSKTLAFKDWAESLEEYKNTNEITTQANYWNTIIAAMQEEKLFIDDDPHDHGFEDVTITFRKDKTEQLIRQAGKAFG
TEINDLLISAIGAATHALTKQSKVSVCLEGHGREAIHKKVDIDRTVGWFTVKFPISVPCHDDLRKSIIETKEMLRKVPLR
GLGFGLLQGDYLAIESDIYFNYLGQLDTESDATLFTNNKSIANSNSTEGAIDMNGYIVQNMLHFIIRYDRSKYNRKTVER
FAALYKEKLSDIITYCCVQDKPVKTASDYSCDDLTSDELVRIQEFIADKEDLDDIYTLTPMQEGILYHYLLNRQSTSYVT
QMVYHYNGNQSEHTFEVALKLLTKRHEVLRTAIVYDHGDKSRQLLLRSREIEYAVLDWSGKDRAEQASHISTWIDSELQR
GFDLQQEALLRVTNIMLSRTEGKLIWSSHHIISDGWSDNVIFNDFIQLCEKLQQGLSVVELEQGLEEEKEKSLNFKDFVN
WLEKQDEERSLLYWRELLLDYEESADIKPLRQPEETSSEMKRINHKLPVDVTRQLLNTAATYNITLNTMIETALGIVLQE
LCGIKDVVFGKVVSGRNADLRGIEHIAGIFVNTIPIRICCDDEMHICDIWRKTQEQAILSDQHGHCSFAKVQRLTKQNTD
LVKVLFTFDNDDSWDEVRQSETSAFRFTVEDGREQTNYTTTFKAYLDISQLHVDVLYDPRKLAAEDIVNIMSKLQNVLLS
CAANPEMKLSEIETTTEEEKEQILGAFQTKPTQYPHKESIKNLFEAQVKKTPDRIAISCGTEQLTYAELNAKANIVANRL
LAAKVRSEAGKGETMVALLLERNMDMVIGILGVVKAGYVYVPIDPEAPQDRIQYTLADSGASLLLVNQGKEHRVSLPQGN
VYAIEQLYSDEPDAAWGAELKADPDVAVAPQDALYIIYTSGTTGEPKGTVIEHECVVRLMHNDDMPFHFHEEDVWSLFAA
YNFDVSVWEMYGALLYGGHLVVIPNDVKKDSYALLDLLREKRVTVLNQVPSSFYNLMQVELAEAGERLYVKHLIFAGEAL
QVGKLKAWQAKYPEMDIVNMYGITETTVHVTYKAITAEDIALDGNDVGKALPTLKIYILDGDKLCGIGVPGELCVAGVGV
ARGYLNKPELTNKKFVANPFVPGERMYRSGDLAQWLPDGNIAYLGRMDDQVKIRGFRIELGEVENALRQLEGIQDAAVIV
RDTHDGEKALFAYLISKDVVQLQAVRRDLEEVLPAYMVPAYMMQIEQLPVTRNGKLDKRALPEIVAVSEKAYTAPKNELE
AQLCSIFSEVLGVERVGTQDSFFELGGDSIKAIRIVSKLRSAGYHIAIKDIMQKYTVEAISYAAQKSALEEQYEQAEVSG
IVPLTPIVREFASWNLPKPHHFNQDMLMEIDLEDEKQLREVLNALTAHHDILRSVYREGQLEIVKVSDSKAYELKVYDYR
EEQAAAELMEAACTDLHSSINLEEGPLVKAAMFRTVAGNLLFLGIHHLVVDGVSWRILQEDISTAIRQVKEGTKIQFPSK
TASYKAWAEALAEYKDSHLLQKERPYWEQVAEQMAAGELRMDEQTGESGYGSCTIHLNKEETAQLIHQASRAYNTEINDL
LISALGMTVRKLTGARDVTIGMEGHGREPIHKRIDIDRTVGWFTSIYPVVVPCHEEIAESIITTKEMLRKIPNRGLGYGL
LQEQLPVRPLEVMFNYMGQMDAEAKGRKLHFFSSGKGSADENIVYRKVNINGSLLKEQLHFSFVYDKSKLSAGKIQLFIE
TFHDCLMATIQFCASQEEVAITISDTDATDLAADDLQVINELFDLE
>lcl|NZ_CP026520.1_prot_WP_042227883.1_2704 [locus_tag=PC41400_RS13770] [protein=non-ribosomal peptide synthetase] [protein_id=WP_042227883.1] [location=3014611..3022764] [gbkey=CDS]
MSSKMVLGSNIENIYSLTPVQEGMLYHKLANGDSTAYVVQHRFELNGLINEQKVEEAIQLVTMKHDVLRTSIVYEKLTMP
RQIILKNRTIEYERIDLSDIAAYDRNNEMDMIAQLDVKRGFDLQRDSLLRVKLVHFGANQYKMLWTIHHIIIDGWSLSIL
FSDFKTYYDMLENGKSLPSIMSIVETDKKRIEKYGDYVKWLEQQNASEGIAYWEQLLSDYDGTADIKSMEQTPLEVETVS
LSEMKLSKEQTASLQNLVATHHITMNTIVETAWGIVLQAYTGAKDVVYGKVISGRHVAIRGIEKIVGPFIVTIPARIRYK
EEDRVVEVLRSTQLQGTKSTAYSYCPLQQIMSLTKQKSELIKYLYAFENYELDETLLTQHSEGLGLKIEAYREQTNYDLV
LIVNNEHGYLRFQMQYASNHYGSREIEAMLLRLQAVLLSFANNPEMKLSELETITEEEKEQILGVFNDTYLPYAKDKTIP
ALWEEQVANRPDATAVVYEDNHVTYDALNRKINQVAWALRRLHIKPDDRVVIVAERSVEMIAGMYGIIKAGGAYVPVDPT
IPAERMRFILNDCKPKAVLVYKTTIETDLPTINLADDKLWEGQGENPEQVNRPDDLAYIIYTSGTTGQPKGVMLAHSGVV
AMRSYLQEKYEVTQQDNVLQFANYIFDASVWEMTLSLLLGAKLTLVSQGVLSDVSSFNAFIKKNGITLTLLPPQYYLQTD
ISGLRVLTTGGSAANAEMIKKAGSGTRYINAYGPTESTVLATGWEYDGESDIPYPLPIGKPTSNTQIYIMDDMKLCGIGV
PGELCIAGSGVAKGYLNLPALTAEKFIENPYGEGKLYRSGDLAKWLPDGNIAYLGRMDDQVKIRGFRIELGEVENALRQL
EGIQDAAVIVRDTHDGEKALFAYLISKDVVQLQAVRRDLEEVLPAYMVPAYMMQIEQLPVTRNGKLDKRALPEIVAVSEK
AYTAPKNELEAQLCSIFSEVLGVERVGTQDSFFELGGDSIKAIRIVSKLRSAGYHIAIKDIMQKYTVEAISYAAQKSALE
EQYEQAEVSGIVPLTPIIREFASWNLLKPHHFNQDMLMEIELEEEKQLREVLDALMAHHDILRSVYREGQLEIVKVRDSK
AYELKVYDYSEEQAAAELMEAACTDLHSSINLEEGPLVKAAMFRTVAGNLLFLGIHHLVVDGVSWRILQEDISTAIRQVK
EGTKIQFPSKTASYKAWAEALAEYKDSHLLQKERPYWEQVAAQMAAGELRMDEQTGESGYGSCTIHLNKEETAQLIHQAS
RAYNTEINDLLISALGMTVRKLTGARDVTIGMEGHGREPIHKRIDIDRTVGWFTSIYPVVVPCHEEIAESIITTKEMLRK
IPNRGLGYGLLQEQLPVRPLEVMFNYMGQMDAEAKGRKLHFFSSGKGSADENTVFRKIIVNGGILDGKLQLVIGYNRSYL
SSDRMQSFSERYMAALLAVKDHCIAIKESVRTPSDYRAADLTRTDLSVIQQTVGDSSQVERIYGLTSLQKGMLYHSIAEP
ESTAYRNQNIFIGQGYADEHMVKQALGLLAIRHEVLRTAIIHKDISMPKQVVIREREVEYERTDLTGVEAVVQAEMVEAI
AHGQIQRGFDLAQDPLLRVHHVVLSKDCYQLIWNFHHIIMDGWCLSTVYGEFNRLYTALQKGALTSDLKKQVIKDKQDQT
SYEDYINWLEKQDQQKGLSYWSDVLAGYEEVAEIKPVYTPQESEQQVQRLAITLTPEDRLRIKELTSAHQMTVSNIVETA
WGVVLQAYSGQKDVVFGKVTSGRQSDVRGIEDIVGLFINTIPVRVASKEGMSVMRLLKEMQQQGSESEQYAYCSLAEIQA
QTEQKQHLIQVLYAFENYYVDEDKLTDFTYEITSSRDQPPFNLTLCAYESGEKIVCEILYNPNVYANEDIVQMLARLEHV
LHALAANPEMKLSELETITEEEKEQILGVFNDTYLPYAKDKTIPALWEEQVANRPDATAVVYEDNHVTYDALNRKINQVA
WALRRLHIKPDDRVVIVAERSVEMIAGMYGIIKAGGAYVPVDPTIPAERMRFILNDCKPKAVLVYKTTIETDLPTINLAA
DKLWEGQDENPEQVNGPDDLAYIIYTSGTTGQPKGVMLAHSGVVAMRSYLQEKYEVTKRDNVLQFANYIFDASVWEMTLS
LLLGAKLTLVSQGVLSDVSSFNAFIKKNGITLTLLPPQYYLQTDISGLRVLTTGGSAANAEMIKKAGSGTLYINAYGPTE
STVLATGWEYDGESDIPYPVPIGKPTSNTQIYIMDGMKLCGIGVPGELCIAGSGVAKGYLNLPALTAEKFIENPYGEGKL
YRSGDLAKWLPDGNIAYLGRMDDQVKIRGFRIELGEVENALRQLEGIQDAVVIATNNHEIQSLCGYIVSHTVVDPFEIKQ
HLKKILPSYMIPSFIMQLEQIPVTRSGKVDKHRLPLPNVRALMGHDTFVAAQNDREKELVTIWRDILASEQIGIHDNFFE
LGGNSLLIMSMLARIEERYPGIVKVGDIFANSTIAQLATFIDVSSQTAMRCAQVPFPDSYLLGSRNKVFVFHFQDNRDLF
HALKEMQSESETALESWLLFLYTYSVIEATGQQELSVCYGANNTFASFEISNHDDLNVLLENVKNKEQAAPKCTQAKFII
DYKQNGLFPMFLYQFTGDKGYTEYCDFSFSFTMNEEAVYFEAKILNRTLNSDYIQSLFSRWIQIVQHVSLKDKIITK
>lcl|NZ_CP026520.1_prot_WP_129112479.1_2705 [locus_tag=PC41400_RS13775] [protein=hypothetical protein] [protein_id=WP_129112479.1] [location=3023211..3023657] [gbkey=CDS]
MLTALNARFRDIFTALEIKTMGEMSYVHINKYLDGEVHHDHSSQQRQTFVTYYRSFVFNVTKWIKSKLSIEEQSYFSKYT
LPDFPFDNRDYDARKNAIESAQKTRKDETSAVAPLLPRIRAQVHFRYNPNKTSTRDYKQTNNRCEKQL
>lcl|NZ_CP026520.1_prot_WP_042228844.1_2706 [locus_tag=PC41400_RS13780] [protein=sugar ABC transporter permease] [protein_id=WP_042228844.1] [location=3023833..3024717] [gbkey=CDS]
MKKFKRWGEPAAYFYVLPAMVPLTLFWLAPMVYIVYLSFTDWDFMSPVKTWVGFENYKELFANEEFYKALKVTLLFGLGM
VIPTMTGGLALALILSRQGRGAGVYRAILFSPWVTPTVAVSIVWSWIFEPRSGLANSILIGLGADGLPWLQSSKWALLCV
IIVSVWKYVGWAMVFYLVALQGVPKDLREAAFMDGAGEWSIFKSVTLPLISPTSFFLFIILWLQSFQAYDQISILTQGGP
SGSTRTLLYMYYQSAFDSFNVGEASAVALVLVFISALFSLVSVTMSRRSVHYLQ
>lcl|NZ_CP026520.1_prot_WP_042227881.1_2707 [locus_tag=PC41400_RS13785] [protein=carbohydrate ABC transporter permease] [protein_id=WP_042227881.1] [location=3024730..3025572] [gbkey=CDS]
MRVGSMIVRPLHHVILAAFSFIMAFPFIWMITSAFKTNDEIWAYPPVLLPDKLLWENISKAWHAAPFDLYVFNSVFVSVC
IVLIQTVNSAMMAYALVHMRFPLGRPLLAIILVSYMLPGAATYLPGYIILSNLGLIDTYGGLIISNSVSVFTIFMIYQAF
RQIPREVVEAAKIDGAPHRRILWTMVIPLTAPYFVVTGLITFIEMYNNYLWPSLITHNPDLYLVSAGLRSFFIEGGAYGM
NWAQVMAASSFTILPLLILFIFAQKFIMKGVSQSYGVNKG
>lcl|NZ_CP026520.1_prot_WP_042227879.1_2708 [locus_tag=PC41400_RS13790] [protein=ABC transporter substrate-binding protein] [protein_id=WP_042227879.1] [location=3025637..3026983] [gbkey=CDS]
MLKTWEKGRILLMTAIVIMFLAACENGNSGKNANNGNVKPVNAETAAKEPVEIEFWYALGGKLGQNVAALIKEFNASQDG
VIVKGVAQGNYEETQQKLQAAIASKKVPAAVLTQNNDWARKGYFAALDDYITVDVDFNKDDFIPAFLQQGVVDDKQFFLP
MYGTTQVLYYRKDMLEKAGLTEDVFNTWETLADAAAKMTVKKGDTTKVFGWMPMWGRENMMDAALSRGGRILSEDGKTVM
IDSTEWIETWEQFRKWIHEEKIMGIHSGGQGWEYWYKTIDDVMQGRAAGYTGSSGDQGDLDFNVVAAHTQPGWANRAPAP
MARALMAGIPALASDKQKQAAYEWLTFFSNTKNTAKWSINTGYISVRSSALEDPDYKAFSEQNPQIKVPLMQAQSASSAF
IDQTGGKIDDALKIAADKVQIENVPAADALKEARQKAQEELDKVTGVK
>lcl|NZ_CP026520.1_prot_WP_042227877.1_2709 [locus_tag=PC41400_RS13795] [protein=CehA/McbA family metallohydrolase] [protein_id=WP_042227877.1] [location=3026988..3028457] [gbkey=CDS]
MMSTVICTEVREASLLPGPSLLTYSFSMKEPMDWISLVITYEKLDWMQVFVKDPGGFIRMQYTGKKRAERILLGRLQENC
SIGSVPGDVGAGTWKAEVIAYARDEDSMFKLEWMHGKGVSAESSTVPIAIGIPWVTLDGDARYEQERLNCGARWYKGDFH
MHTSLSDGKQSPEQLMESSLRQQLDFIVVTEHNHRHTAWPSHDRLLALPGMEVTATQGHWNVLGITDWLPLYDEKGTLTM
EDAEGMNRIIAAAREQGAVCSLNHPYLAPWDWRFGRAELSLFHAIEIWNDPTYEGNAEAAERALMLWDACWARGLRLAGI
GGSDTHLLPHESYMEGGPPSLVGDPATYVYCDELSEVQLLDAVKKGRCIVSRGPQLEPRIYSGGRQILPGDQVSIDAQAA
VPIRYSIAVKGIKEKGKLIWLLNGLPVFEAVVNEDGPYVYETEWTGEAYRWLRVELRSEYGTLLAFINPIYANPIGCSSL
TWEEVNAEL
>lcl|NZ_CP026520.1_prot_WP_042227876.1_2710 [locus_tag=PC41400_RS13800] [protein=HAD family hydrolase] [protein_id=WP_042227876.1] [location=3028454..3029203] [gbkey=CDS]
MSIRIQCGASSWEVDGVLFDKDGTLLDFAYLWGAWSSLLMDRLRHRALGVNWETVLGLCRDGEKFHYDPLGPLAMGSNRE
IAALLAGHLYLSGLAWNEAITAVLTDMKHVSTQIDELRPVRAMPGVFSFVQQLSREGVPIGIVTSDDTEAAVKHLEWLGM
RKHFQVILGSDGHVNGKPHPDLAIAACAKLQIAPERTLMFGDTKADIQMAREARLAGIVRIGLERTQASDGEEEQISHYE
QLQWTTLTS
>lcl|NZ_CP026520.1_prot_WP_042227874.1_2711 [locus_tag=PC41400_RS13805] [protein=extracellular solute-binding protein] [protein_id=WP_042227874.1] [location=complement(3029438..3031609)] [gbkey=CDS]
MPTIKDIAKAAGVSHGTVSNVLNGRGNVSLEKINLVEEAARKMGYQIHARAKSLREGSTTTVSLILPNISSDSYNKLYIA
IHSTLLRVGYTTNLYLTQDQKENEQALLQEIAANRDCAVITVSCLDDAMEYYDLLGIPRENIIFVYRKLQNTMKFASFDA
DASGRAIAQKIKEKAHRKIGIFTDLLRYSCDRDFRNAIMEELSPQDLRIHTIAAAQSTAYNHAFDFFAGDGFDCIVTSDL
ERARYIKNAHFLGSTGQCPEIYSLADSEFLTEDFIHKYHINYQWLGKEIAETILRHETTDQRDGAPIIGLNRGFMFEDQA
LGHPTANKTLNILTIPSPTTDALKKLAPHFKKTTGIDLRLAVYSFEEIYDILSDIDNHKHYDIIRVDMAGLSWFAGRTLK
PLSPEMRELLRNYPPQLIDRYAYVNEIPYAVPFDPSIQMLFYRKDIFEDTKVRRMFYEQYKAELRTPSDFEAFNRVSAFF
SRASNPESPLSHGSCVTTGNSEIIASEFLVRYYAEGGHLIQEDNKLALFKEIAINAFHSYIDHLSIAKNLPAKWWDESIS
LFEDGQIAMLIVYMNLFSNIAHSEISPLVGFAEVPGQQPLLGGGSLGVSKYCRKDDEVSAFFRWFYDHEIAEQVTLLGGA
SVNDFIYSNQNVMDQYPWLKFAKDLEYRGVRESAFPTGHNFNLRKVEKIIGLGVKNVMNQMMSIEEAIDYINARLSEKEF
LAL
>lcl|NZ_CP026520.1_prot_WP_129112562.1_2712 [locus_tag=PC41400_RS13810] [protein=MgtC/SapB family protein] [protein_id=WP_129112562.1] [location=3031953..3032453] [gbkey=CDS]
MFGHWEMYVRILASALMGLFIGWDRSHKHKPAGMKTYTYTSVACTLITIISISTASHYNEAGQLMRADPMRLTAQIVSGL
GFIGAGVILKDGERVKGLTSAAMIFFSGGVGIGIGAGYYGIVIFSVVLTFFLARISRFLPMNNPGDDRKSRSGNDAARKQ
QEKECI
>lcl|NZ_CP026520.1_prot_WP_042227871.1_2713 [locus_tag=PC41400_RS13815] [protein=MBL fold metallo-hydrolase] [protein_id=WP_042227871.1] [location=3032492..3033310] [gbkey=CDS]
MKVHFLGSAASEGIPNPFCKCEVCEKTRSEKGKNIRTRSAAIVDDTMQIDFPPEYSYQVMRDDVGVTRISELLFTHTHPD
HFNAGDLFSRMEGYGHHVTHPLHIFGNDRAVNGCLDVLSGYSPERFQFTRMIPFVTVESSGARITPLLANHAKWETCLIY
FIEKEGKTLLYGHDSGWFPELTWSWLADKHIDLTVLECTHGYNQNPHSDGHMSIETVLETQKRLLDSGIHSATSQIIVSH
LSHNSGFMHDELVDIFKPHGIEVAYDGLIKFI
>lcl|NZ_CP026520.1_prot_WP_042227869.1_2714 [locus_tag=PC41400_RS13820] [protein=MFS transporter] [protein_id=WP_042227869.1] [location=3033326..3034576] [gbkey=CDS]
MKTTIVPRLLIMMFLQYFVQGAWNMTLGLVLSTSGLSSIIGTSYALLGIATILSPLFIGMVADRFIASQKIMGILHIVNA
CVLLAIPAQIQSGNATMFLVLLFIVGVLFYPTTGLSNSISFRHIHGAKTFPVIRVFGTFGFMVIGFLLGQMGFSGSIMTF
KIAAISAIVLGIYCFTLPNTPAAGKGKKFLIKDLLCLDALALFKDKNFTVFMGCTIFLMIPKTAYSAYVPVFLGALGFNN
AATIMQIGIATEVIFMFLLSLFLFKFGFKKIILWGAITWVLRSLLFSFAAVDAGILVIVIALALQGICWDFFFTAGDIYV
DNKADVTIKAQAQSLKFMISNGLGLLFASSVTGYIFNHTVVDQGAAALKQWQQFWIYPAVVAAVVALVFFLFFKDKGMIF
QKKNAEEGAVSELKAV
>lcl|NZ_CP026520.1_prot_WP_241688864.1_2715 [locus_tag=PC41400_RS30290] [protein=hypothetical protein] [protein_id=WP_241688864.1] [location=3034762..3034932] [gbkey=CDS]
MIQRNFQSPNLKERLEKKNVNVYSTGLNGNIPLISDGFKIEVLTEKDGLAPKKSKP
>lcl|NZ_CP026520.1_prot_WP_042227868.1_2716 [locus_tag=PC41400_RS13830] [protein=STM4015 family protein] [protein_id=WP_042227868.1] [location=3035248..3036093] [gbkey=CDS]
MSEVKLSIDYDQYEDGKKMKDLLEELANSPQSGELTSLIIGDWGGAYENDSEETVETLVALKGRFPKLTSLFIGDMSFEE
CEVSWIMQTNLAPLLEAYPALRSFTIKGSTGLDLNPARHENLEELTIICGGLGKDVIHQIYEGSFPRLKKLELYLGVDEY
GFDGGLEDVLKLTEPGKFPELTYLGLKNSEIQDELAIALADAPILDGLHTLDMSDGTLSDTGAEALLNSEKVKKLQYLDL
SHHYMSDEMVNRWKSSGLNVNTDDQQDMDEDDDYRYPSLTE
>lcl|NZ_CP026520.1_prot_WP_042228843.1_2717 [locus_tag=PC41400_RS13835] [protein=STM4014 family protein] [protein_id=WP_042228843.1] [location=3036111..3037478] [gbkey=CDS]
MILIGNPDNRRTAGLQNARAKLGLPPAVVLPYAGLLDGTASLAQAASLAGAPSQPAGEPAPLLRLDAPGELFAVERGLIA
LGAPDHEEAAHGDRLLPFGHRSDPQPLSVRSALALAEQEGRQYHPSQWFRGFARLLARLRREAVELWPQPQWTNAPEDIA
GMFDKRQTHRVLSAAGVPVPRLLAPPEEIAGYEALRDLMLSRRVHRVFVKLASGSGACGVVAYRINPRTGAEQAVTTVGV
ENYGARAPVFYNAMKLRRYTDESAIRQVLNWLLRHGAHVEQWIAKAAYGDRTFDIRQLVVAGEACHSIARVSRTPITNLH
LRSERVSVDDLGLAPGIRAEAAACAVKALAAFTRSLAAGVDVLLDSATHAPYILDMNPFGDLLYRVVHKGMDPYEWQMSR
LTGGTAEAAEESSRSAASGAADPTGSAASAGFSEAGNSGKPGGAAGSPGEGGTLR
>lcl|NZ_CP026520.1_prot_WP_042227867.1_2718 [locus_tag=PC41400_RS13840] [protein=STM4013/SEN3800 family hydrolase] [protein_id=WP_042227867.1] [location=3037475..3038257] [gbkey=CDS]
MNMNEIVGTHDILMITLDTLRYDAAVLEEANCPNLCGGGSWEKRHTPGSFTYAAHHAFFGGFLPTPATTDKSQHIRLFHS
KNTGLKTHPHTWLFDTPDLVSGLQEEGYRTICIGGVIFFTKKVPLARVLPGYFQESYWRMTFGVTNPRSTEHQVNHALKL
LERTEQSEKLFMFLNVSAIHGPNHYFLPGAKRDSVESQRAALRYVDGQLGRLFEAMRERERPVFCLAFSDHGTCYGEDGY
EGHRLAHEAVWNVPYREFVL
>lcl|NZ_CP026520.1_prot_WP_241688897.1_2719 [locus_tag=PC41400_RS13845] [protein=STM4012 family radical SAM protein] [protein_id=WP_241688897.1] [location=3038332..3039684] [gbkey=CDS]
MSVSAWEENLRSAPYRSYLYSYPHKTAYRELDRKIPLQELWTDEPAETFFLYMHIPFCGARCGFCNLFTLPDKRTGAHEQ
YVDALERQAREWAPHVAHKKFARFAVGGGTPTLLEPALLDRVFRIAEDAMGLDSGEASISVETSPETVTAERLDILQAHR
VDRVSMGIQSFVDAEAAAIYRPQMPLVVEKALEQLKRYKFPILNLDLIYGLPGQTVDTWLYSLERALAYEPEEIFIYPLY
TREHTIVKPEEIRQRGADIRMACYTAARDRLLAAGYTQYSMRRFARASASSPKTLLPYGCQEEGMVGLGCGARSYTQNVH
YATRYAVSRSATAGIIADYVAAERHDLIDYGFVLDEGERKRRFILKALLHREGLELASYARRFGTKALEDYPELNRLTET
GLAEHVPGTGAGEDGSLFRLTDDGIALSDSIGDWFISPETRARMEGYTFR
>lcl|NZ_CP026520.1_prot_WP_042227864.1_2720 [locus_tag=PC41400_RS13850] [protein=STM4011 family radical SAM protein] [protein_id=WP_042227864.1] [location=3039681..3040598] [gbkey=CDS]
MKATLYYRGSLSSCNYDCPYCPFSKNKDSAATLAKDRRQVETFTDWAESQGAEGHRLSVFFNPYGEGLTHRWYKEAMVRL
SHMEHVDKVAIQTNLSAGLAWTGELNPKKTAFWVTYHPGQTTEEKFLEQVLQLYRRGIPFSVGSVGVKSAFGRLESLRRS
LPSDVYMWVNAFKDKRDYYTDEDRTFLRSLDPLFDLNAEDYESLGRACGTGFDVFYVQGPGIVKRCYKDRRVIGHLYRDG
LDALSQPRPCGMDCCDCYIGYIHMPDLQLRKRYGDGLLERIAGAGSQRRMPRVYEEGVESCDAHA
>lcl|NZ_CP026520.1_prot_WP_042227863.1_2721 [gene=cysC] [locus_tag=PC41400_RS13855] [protein=adenylyl-sulfate kinase] [protein_id=WP_042227863.1] [location=3040582..3041115] [gbkey=CDS]
MTHTLNKTAPAGLAYWLTGLSGSGKTTTAKALAAELAALGHAVECLDGDELRRQVGTGLGFSREDRMENVRRAVFISGLL
NRNGITTVVSLISPYREMREYARQNLPRFIEVFVDCPLEECERRDVKGLYAKARNGEIPAFTGISDPYEVPENPELVIRT
NGDTLELNVRRLLDFYV
>lcl|NZ_CP026520.1_prot_WP_042227862.1_2722 [locus_tag=PC41400_RS13860] [protein=GNAT family N-acetyltransferase] [protein_id=WP_042227862.1] [location=3041207..3041644] [gbkey=CDS]
MEIRKPIPEEIQQIRRLSPQALSEGTLGKAEPDRDKVRQLVDPLLKKGCYYLIAAEEKRLAGWVLIGESKDSFTDRVIGF
VYELYVLEHFRGRGVSRKLMQEAVEQLKSLGYPEVRLSVYAGNHALHLYESLGFAPRTITMTLPL
>lcl|NZ_CP026520.1_prot_WP_042227861.1_2723 [locus_tag=PC41400_RS13865] [protein=helix-turn-helix domain-containing protein] [protein_id=WP_042227861.1] [location=3041963..3044248] [gbkey=CDS]
MIPIPRPRSFRIALFWRHFANYFVVILIPVIVACTLAHFLVVSLIEKDAQKLNRVIMSRFSEQTDTAFNSLKTNMINMLS
TSNIRSLLKEAGDLSGENTQRMELIHALREQLIKLQSDELVTKAYLVFAHHDLVIDVDTYTDKSYYFDFRYPMGAEEKAA
YLAKFTNKKMMEFTEPKPDGSALSALMSYPFNTNVPEVYLVVDVSRDKLQRLINIRENWVAGTAIVGSRGEIISSNGVVS
DGISRIPSPLAADSGESKFLITNDKAISLVKSGFDDTSYYLSLVDLPTLMKPAHLTRLISWIFLAFFIVVGSFVSYYLSR
RMYRPILEIKEGLKSHRSAQVLQMRSDGNDYDVIKRFSQLIISENKQLNQRVTGMFPIVQEHFVTKILLGEYRDALSIET
YAGEIGFTYTKKAARTVLGIALHYDPDVYGRLSETSKSFLMAELKERILQLVPSTIWLCGIKSDLLVCVIQKDPFLSRSP
EEEANLIRAALQLYGSYFKATIGIGRTVYGIEELHLSYEQAVSVLQRRGLNANVEICSGQSALDSTQCDTFLSVQEVGRI
LNRCKTRDYDKLLQSAYEMVEEAVRKNATAVQIKYLCTDILNTWVRAVETERKDFNVPFYAGLFDKVNACMTLEEVKRCF
REFHELLFQPAKGDERAGKFTDILAYIHEHYDEELSIEQFANSMNMSVGHFSRTFKEEVGEKYVEYIAKYRLVKAKELLL
QTDLKMDDIAEKVGYWGRNSFIRMFRKYEGITPAKYRTIHQ
>lcl|NZ_CP026520.1_prot_WP_042227859.1_2724 [locus_tag=PC41400_RS13870] [protein=ABC transporter permease subunit] [protein_id=WP_042227859.1] [location=3044437..3045357] [gbkey=CDS]
MTKSARFWYLVRKHKALYLLMLPGILYYIIFKYVPMYGIIIAFQDFSIGRGILESKFVGLKHFIEFFYVTPDSWKLIRNT
VMLNVYDLLFHFPAPIVLAILFHELRNKWFKGFVQSVSYMPHFLSTVVIAGILVTFLSPTTGVVNHFLVKVFGAEPVMFL
GIPEWFRTIYVSSEIWQKIGWGTILYLASIAGIDPTLYEAAKMDGANRFQQIRHITLVGMLPVMIVLFVLTLGHFMETGF
QKILLLYNSMNYETSDVINTFVYRRGILDSDFSFATAVGLFQSIIGLILVVTANRIVRKYSETSLW
>lcl|NZ_CP026520.1_prot_WP_042227858.1_2725 [locus_tag=PC41400_RS13875] [protein=carbohydrate ABC transporter permease] [protein_id=WP_042227858.1] [location=3045371..3046231] [gbkey=CDS]
MKIKTSLGSRVFDTLNIILMLLLVVVMIYPMVYVFSASISNNALVASGEVLLFPKQVTLVAYERLVTNPDLWISYWNTIR
YTVLHVALTLIVTAAMAYPLSKKWLPGRRTILLLAAFTLLFSGGMIPTFLVVQKLGMLDTVWAIVLPSLISTWYLFIMRT
FFEALPEELEDAATIDGCSSFQVLLRIVLPLSMPVMATIGLFTAVNQWNAFFDALIYLNDRSMYPLQIMLRNILIAGTSV
QGEGDLNHLETLKYAMIMIATLPILCVYPFIQKYFVQGSMIGGIKG
>lcl|NZ_CP026520.1_prot_WP_042227857.1_2726 [locus_tag=PC41400_RS13880] [protein=extracellular solute-binding protein] [protein_id=WP_042227857.1] [location=3046493..3048055] [gbkey=CDS]
MKKRPFAVILTVALLAGVVAAGCSSKDQGASPGDGGGSDAKKPVKFSWLAYDRVEGKVRQDWEIFKEIEKKTGVSVDFQI
VSQEGLTEKRQIMIATNSVTDFIQVSTQEGREHGPDKVFLNLKDYLDKAPNLKAFYEKYPEAKAVASGADGGIYTVPVIE
GDAEGKGFNFIWYTRKDLMDKYGLKPPTNLDEFYQFLKSYKEKEPSSYPLIPNAIVGDTGLYTVFGRAFTGIQGFFNIDP
TTDKYAFAPYHKGYKDSLVFLNKLYTEKLLDPEYSLLTQAQWEERILKGKSLVTYFWKADLESLTDKGRKAGTSTFELDA
IPQFAAEGIKNYQFSRPVIGAVGRAISAKVKDKDRAVQFLDYLVSEEGSNYLSLGIEGKTYTMENGKPVYMKEFGASPFN
TLRKDWGVWYDLITLDNAKSREVWERGLSDKSKDINKRYEPFIVPSPKQIVKTEEELELEKSKLNNLNKFLEQKLTEFVI
GKTPINDTTYQQFLDQVKKMGADDLLNMYNTAYTRTYGKK
>lcl|NZ_CP026520.1_prot_WP_042227856.1_2727 [locus_tag=PC41400_RS13885] [protein=amidohydrolase family protein] [protein_id=WP_042227856.1] [location=3048104..3049168] [gbkey=CDS]
MAAAEKLKVIDTDVHNEQNDSALIPYLPEPWKSRVAATGIGYSGSGYYSPIGVMKKDSIPPGGDKAGSDPDFMIKQLIEG
YNVEYAVLTGVVYNISTTHDPDYAAAICSAYNDFLIAEWLGKHKAFKGALAVSTLDPQLAAREIDRVGDHPDIVEVIISS
AARWPLGHRFYHPIYEAAERHGLPVAIHPGAEGGGSSTAPTAAGYPARYIEWHTCLSQMFMAHLVSLVCEGVFAKFPRLK
VVLVEGGIAWLPPLMWRLDKNYKALRSTVPWLTKMPSQYVAEHCFLTTQPIEEPDNPQHLVDLFNMFDAENMILYSSDYP
HWDFDAPDMILRKLKPEARRKIFYENAKKLYNLT
>lcl|NZ_CP026520.1_prot_WP_042227854.1_2728 [locus_tag=PC41400_RS13890] [protein=Rieske (2Fe-2S) protein] [protein_id=WP_042227854.1] [location=3049187..3049543] [gbkey=CDS]
MAVHDVLDAADVPEGGGVLVEVEGREICIYRIGGEFYALHNYCPHQGAPMCAGLVSGTNLPSDVYEYDYSRQGEIVRCPW
HGWEFDIKTGKSLFSDRTRVKKYELEIKDGRIGIVLGK
>lcl|NZ_CP026520.1_prot_WP_042227852.1_2729 [locus_tag=PC41400_RS13895] [protein=DUF6376 family protein] [protein_id=WP_042227852.1] [location=3049821..3050264] [gbkey=CDS]
MKIRLVLAVLTVLMLSGCSLLEGVNSTLNYGEEAKTYLQEAAAFAENLPAAARDAATDPQAGEKVKKELEAMRENISSFN
GLDAPGFAEDLHKQIVGYNETMLGTINGHLEQLNSQVLDYQSLLQSPVIQQVNEMKQLTEKIQNLGQ
>lcl|NZ_CP026520.1_prot_WP_042228842.1_2730 [locus_tag=PC41400_RS13900] [protein=AraC family transcriptional regulator] [protein_id=WP_042228842.1] [location=complement(3050427..3051209)] [gbkey=CDS]
MIKLTNIRHDRLAEWYEESGPGGNTAASLIMLSYGRCLYWIENDKIMLEKGDLLLIREGVTYYGKSIPTFLHNKYVASFH
LTDTGHELPLLSKPWIKIRSGMAEFLLERTRKCYSDWKENEPYAAIRGQAILLEMLALLSREADQAAVSSETGKHAERMK
QYIRDHYREKVTKDELSEFISKSPNYTAALFKKVTGQTISEYVHSVRVKKAVYMLMDSLLTIAEIAEFVGYEDVSYFQRI
FKKTTGRTPSSYMKERPPRN
>lcl|NZ_CP026520.1_prot_WP_042227851.1_2731 [gene=pyk] [locus_tag=PC41400_RS13905] [protein=pyruvate kinase] [protein_id=WP_042227851.1] [location=3051323..3052738] [gbkey=CDS]
MRKTKIVCTMGPSCRSVEVLKELIAAGMNVARINMAHGELEDHSLQISRIRQAAKEMGAIVPVLMDIKGPEVRIGLLKEA
SCEVKTGESLTLTSEEIAGDSTRISVNYAELPQVVKAGDKILIDDGLIELRVVSAAGTEVSCEVINGGVIKPRKGVNLPG
VHTTLPGVTERDIRHIEFGIREKIEIIAMSFVRKAEDVLTVRQLLEDQGCGHIQIISKIENQEGVDELEAIVEASDGIMV
ARGDLGVEIPVEDVPVIQKVMIESCNRAGKPVIVATQMLDSMQVNPRPTRAEVSDVANAVLQGSDAIMLSGETAAGKYPV
ESVRTMATIAKKAEGLYMAEASRRQAGVPAEVNVTEIISQAAVSASEAIGAKAILTPTMSGYTPRMVSKHRPKAPIIAIA
PSDEVLTKLCLLWGVFPVKGEFVQTTDDMFKAAVSSGTAQGLLQSGDYTVITAGVPIGKTGTTNLIKIMQV
>lcl|NZ_CP026520.1_prot_WP_042227850.1_2732 [locus_tag=PC41400_RS13910] [protein=PepSY domain-containing protein] [protein_id=WP_042227850.1] [location=3052908..3054770] [gbkey=CDS]
MKTAFLLVFLLAVNTISTSVPESASGISSKQALSIAQSHVSDRKAEWKVGLTQYPPELFGEEAGQSGSSAARTEKRKVWK
AEAKFESGNKEVYYIDASSGKLLSLSEIEASESGLRELPVALEARQLPHPSFQESGVSASETTGVSSSLNVTLEEKKGGF
RLKLAEEKNTTATVMQSLVSPDSGKTVLHLVRRTSPAEGGIRYRLDLVAMQPAKKTFKVFSFADLQMGDTYSTDTLAMGY
GFGDDSHVVMIQPVGKAGGAGIGYRIVSLDVGSGRLRELAANVPPDLSPDFLASGWLNADGSKLYLNSYRGGKLWIADLA
GGGVKALDGTFSHTWPVFRLFKSPDGERFWYEKDGYRLYDGKGVKLADLPADTGLHSNPAFKWSPESRYAAYDYTFTKSD
ENVIGGEDRKVIAPQGIKIFGRDGKPVWSTDARELKHGDRLELRGWIGGSGESIGSALSAGSTANAERSSSPNAAILRSY
TLERAEDREPRKTDSVYHLVQPATGKLTKLSEAKLAELHNPQPVLGQSGALMLVDAGSALYTLVGDETNSARPVKLLPAS
ESKLLWAETDYAKGSVTITAYDIASKKVRTTETKDLSGGDLQAAGEEYVIDSELVYQKLQ
>lcl|NZ_CP026520.1_prot_WP_042227848.1_2733 [locus_tag=PC41400_RS13915] [protein=ATP-dependent DNA helicase] [protein_id=WP_042227848.1] [location=3054912..3056828] [gbkey=CDS]
MRNYPFAFDPTKPFIQQASDWIADVFYELLPEAGFEVRDEQIYMAFQLERAYQEKQTIFAEAGVGTGKTLVYLLYALTYA
RYTRKPAIIACADESLIEQLVKPEGDIAKLARHLDLTIDARMGKSQHQYICLNKLDDTRGGGSDDEEVFSEIYHGLPSFV
HYPDTLQSFYPYGNRKEYPELTDEQWERIGWDVFQDCLVCSQRHRCGQTLSREHYRKSADLIICSHDFYMEHVWTYEARK
REGQLPLLPEHSSVVFDEGHLLETAAQNALTYKLKHAVFESIVIRLLEGEIRETLALAIEEAIVQSETLFRVLDESSYSV
PASDRKEFAWSEELKREVSRFRQLIDVIEEELVFESGLSTLDDYQLKIVEEHLEMIQLALSLFDKPGQLISWVTEDSDGI
TLVIMPKMVKEILRDRVFSKKMPVVFSSATLSVNGSFDYMAGSLGIESFLSFSVPSPYDYSTQMKANLYPDKPEEDKTGR
VLELLKQSEGRALILFPSREALGTFRGSLALYPESADYRFLFEGSAEISHLISSFQNDEESVLCAVTLWEGLDIPGPSLS
HVIVWELPFPPNDPVFAAKRKDSAQPYEDVDMPYMLLRLRQGIGRLIRSRDDRGMVSVLGLDLDNPAIREQIRGLLPD
>lcl|NZ_CP026520.1_prot_WP_042227847.1_2734 [locus_tag=PC41400_RS13920] [protein=iron ABC transporter permease] [protein_id=WP_042227847.1] [location=3057231..3058403] [gbkey=CDS]
MVSPKYIKDRGEDLQSVCNKESVREIGDARSASAKKYGKARALTAFPLLLGGLSLFLVISVTAAVMLGSVRIEPVTVWKI
ALSHMPGIGSLIERDWTKAQEQIIWNIRFPRVLLGLVVGAGLSAVGVTIQALIRNSLGDPYILGVSSGSSVAATLVLLFG
GFSFLGAYALPVSAFIGSLVAMALVFTFAKVGGRISTTRLLLAGTAVASMMSAFTSFIVTMAPNAQGIRSVVYWLMGSLA
GAKWEYLTAPALIVVSGILFLLFQHRSLNALLVGEGAAGTMGVNVPAFRTCLVIVTALITGTIVSVSGSIGFVGLMIPHI
VRLVLGSDHRKVLPVSVLFGAVYVIWADVLARLALAPEELPIGIVTAICGGPFFIWLLRRSSYAFGGTSK
>lcl|NZ_CP026520.1_prot_WP_042227845.1_2735 [locus_tag=PC41400_RS13925] [protein=ABC transporter ATP-binding protein] [protein_id=WP_042227845.1] [location=3058400..3059206] [gbkey=CDS]
MNMEANHLSLSIGAKKILDSVTLNVREGEFIGLIGPNGSGKSTLLKTVYRELKPDAGQVFLDERELLNLPYKKAAELMAV
VSQESSVEFDFSVADMVLMGRYPHKKWYESDTEADRGICLRALETVGLGHFGERSFATLSGGEKQRVLIARALAQQAKFL
ILDEPTNHLDIRYQLQIMELVRSLGITTLAALHDLNMAAYYCDRLYIISEGRIAAEGTPAELLVPAVLRDIFGVHAEVNL
HALTGKPVITYLPESFYNEGLKQTEESF
>lcl|NZ_CP026520.1_prot_WP_042228840.1_2736 [locus_tag=PC41400_RS13930] [protein=ABC transporter substrate-binding protein] [protein_id=WP_042228840.1] [location=3059212..3060219] [gbkey=CDS]
MKSLKLSLFVCLALVCMAAISACSAADSTKELVPRGAAESAGTPAAAVELDNMGQKLHFSEAPKRAVSLNQHATEVMLAL
GLEKNMAGTAYLDDQILPELQEAYKKVHVLAKTYPSKEVFAEAAPDFAYAGWKSAFTEKSLGTREELAKQGIQTYVQESS
NKAAPTLEDVYQDILNIGRIFRVEAKANELVDGMKQRLKDIQEQIGTVGTPQKVFIYDNGEDKAFTAANTYITSLINMIG
GKNIFDDIQKSWVEVSWEEVVSRSPDVIVIMDYGDTPVDKKKELLLGKRELADIPAIKNKRLLVLPLSAASEGIRAPIAL
QTLAEGLYPDKVHKK
>lcl|NZ_CP026520.1_prot_WP_042227843.1_2737 [locus_tag=PC41400_RS13935] [protein=hypothetical protein] [protein_id=WP_042227843.1] [location=3060445..3061362] [gbkey=CDS]
MNKKWLVLGSVIGISSLMMVTTGISAMAGTSGYDAYKSALKNTKTVQSVTVQAAGSLQDNGNVALSANGTFKMSREGGSA
SGSGEISASGVKQSFQFYKQGDQAIFKPGSGDVYYVKDAAAEAHKRFKGSADKEMPQQVETVVDALVGNLKDYVAIDVKE
DGSKRVSAELDGAQLPAVVQAIAPIVIKQGAKEHGEPEKKKADTAANPFGNDFLKGAAPELTDAIRLEKVAVNATISPSN
YIQHQEAELTVSGKDAGGKTHQVVVHLQADLSGYDSTTPDTVDLNGKNVQPIESKHRNKWQGKNG
>lcl|NZ_CP026520.1_prot_WP_042227841.1_2738 [locus_tag=PC41400_RS13940] [protein=ABC transporter ATP-binding protein] [protein_id=WP_042227841.1] [location=3061452..3062189] [gbkey=CDS]
MQPILEVSSLTKLYRGSGRGIREVSLRLEAGDIYGLLGPNGAGKTTLLKLITGLIRADGGTVTLFGVDRSLHHEQAMANV
GCMIESADFPDFLTAWQYLRQSARFYPQVTEQRMEEVLDTVGLLGVRQEKIKGFSAGMKQKLALAAAVLPRPKLVLLDEP
TNGLDIEGTVMFRSLIRRMSAEEGTSFLISSHMIHELEQLCSRAGVLAGGSLKIEADVSGLVLEGGTLEDFYMKQIREKG
ELQGA
>lcl|NZ_CP026520.1_prot_WP_042227839.1_2739 [locus_tag=PC41400_RS13945] [protein=ABC transporter permease] [protein_id=WP_042227839.1] [location=3062182..3062928] [gbkey=CDS]
MHRWKAGYVNELYLLLYRKKAGIFGAVSILLPILFAVSLQALEPVLGLLAVGGSFPIQMLGFYTALWIPLFILTLTGDQF
PHEVASRTLKLAFLRPHTRFHIFTAKVAAQGTAVAVLLALLFAVTLLCAMFTGTPLGLAEGLSAAKAYAAAFASMLALSA
VFVFAAQFFKSASGFVVFGIALYAAAKLAPFFVQSFPAFSPVSYTDWHLLWLNDGVPAGRRITASLFLLSSSLLFFAFGY
YKFDRKEV
>lcl|NZ_CP026520.1_prot_WP_042227837.1_2740 [locus_tag=PC41400_RS13950] [protein=HAMP domain-containing sensor histidine kinase] [protein_id=WP_042227837.1] [location=3062943..3064421] [gbkey=CDS]
MSIRFKLLLSYAAMLVIPLVMMLITAMLLVVVFRGDLQSIKDQYGEGEGRFENHGIERLLKEIKRTTEIDPAALTDASYL
AEIDGELQRNQSNLILRKGDEIVYAAASFPYPQLLKNLPAYEKPGYVKMSKSERIGKQLVEVVHADFNFEDGQKGSLFVV
TAVSPFVTFARKFFPILFIALIVILILTHTLLTYFVSRSIIKPLNTLKAAMKRIEAGDLDSRVVPSGRDEIGELSAAFEQ
MRMKLKASIQTQLQYEENRKELISNISHDIRTPLTAIRGYVEGLGDGIADTEEKREKYIGIITTKAEEMDHLIDELFLYS
KLDLNRVPFNFERVDLYAFIEDLSSELEFELGKQGIRYSADIRLKPHTELAIDRDKIKRVFVNIVDNSVKYMNKPDKQIG
LKVLGTGEEVRIELSDNGRGIDPESLPHIFERFYRADPSRNSDTGGSGLGLAISKQIIDGHGGTIRASSLPDEGSCITIV
LPIRTNKESDSV
>lcl|NZ_CP026520.1_prot_WP_042227835.1_2741 [locus_tag=PC41400_RS13955] [protein=response regulator transcription factor] [protein_id=WP_042227835.1] [location=3064418..3065104] [gbkey=CDS]
MKSILIVEDERLIAELQRDYLEVHGFGVDIAEDGENGLRLALTGNYDLILLDLMIPRIGGFEVCRKIREEKDIPILMVTA
KKEDIDIIRGLGLGADDYITKPFKPAELVARVKAHLSRYERLIGPLKSKEEVSIRELVIQLDSRRVFLRGSEITLTTKEF
DLLWFLARNPNRVFSKDHLFEKLWGMNSLGDSQTVTVHIRKLREKIELDSSNSEYIETVWGAGYRFRL
>lcl|NZ_CP026520.1_prot_WP_042227833.1_2742 [locus_tag=PC41400_RS13960] [protein=KGG domain-containing protein] [protein_id=WP_042227833.1] [location=3065345..3065638] [gbkey=CDS]
MARNDDNKMSREEAGRMGGETTSKNHDKDFYQEIGQKGGEATSRNHDKDFYQEIGQKGGEARGNNGNNGSSGGDGKMSRE
EAGRKGGEARARQRDRN
>lcl|NZ_CP026520.1_prot_WP_042227831.1_2743 [locus_tag=PC41400_RS13965] [protein=MFS transporter] [protein_id=WP_042227831.1] [location=3065944..3067188] [gbkey=CDS]
MNVHASQVYLRLKFGLALANSVMFTTYAVYFIQTLGLTPLQLLLVGTVLELTVLLFEGITGVVADTYGRRKSVILGMFVM
SGSFFLEGSVIGAGAVVPVISLFVWVLISEAIRGFGETFLSGADTAWFVDEVGEEHAGPVFMRAKRWTLTAALFGIGISV
GLSTLAANLPFLAGGLLYAVLGVYLLLHMKETKFVRQERPPGSGPWTELKSTWLTGAREISSRPILLMMLAVTVFSGAAS
EGYDRLWEAHLLKEIKFPAELPISAALWFGLISVLTILLSMLAMWIAEKRLDTTDERIVFKSMFVLTALRIAAVLSFAFS
PNFGWALVSLLAVGVIGTVSAPLFETWLNLNIESRARATVLSMMSQSDALGQTAGGPFVGWIGNRLSVRASLTTAAVLLL
PILFVFGRVLRKRQ
>lcl|NZ_CP026520.1_prot_WP_042227829.1_2744 [locus_tag=PC41400_RS13970] [protein=LysR family transcriptional regulator] [protein_id=WP_042227829.1] [location=complement(3067333..3068226)] [gbkey=CDS]
MNLEQLEFIVEVARAGTLTQAAQNKHMTVAGISRSISFLEKELGVRIFIRSRTGAVPTPEGQMILGKASAILSQVHQLQA
EANSYGKLQNARLRVATIPGPVSLLVDTLTELKKDFPGIRLEMIEKGTKEIIDDVRHNRVDIGFVLFPDTILENACLHAE
PLVEGRMVVGVSRHSALAAAKSLSLNQLAAHQLVLYHDEYIARFIATLGDSVNVLFTSNNIDAIIKAVREGIAVTLGTDY
TFYGNPDYENGDIVMVDAELPGEQPSYLWCVHPKEQPYSQAAHLFVSRLKMQLKRRR
>lcl|NZ_CP026520.1_prot_WP_042227828.1_2745 [locus_tag=PC41400_RS13975] [protein=SDR family oxidoreductase] [protein_id=WP_042227828.1] [location=3068389..3069141] [gbkey=CDS]
MTTTQRVAVITGGASGIGRQTCLKFARKGDRVVVADFNEAAGMETVEHIKKEGGEAIFVKVDVSSQESVQALVDEAVAKF
GRIDVMFNNAGIGGAGPVLTQNMDLYHKTIAVNQHGVAYGIMAAGNKMKELGIKGVIINTASVFGFLASPGTFAYHATKG
AVIMMSKSAALELAPHGIRVIAIAPGAVDTPIIQGFKDNGMLESMKAKVMGGELTKPETVANSVYLLSLEEADAINGSVV
MVDQGYASFK
>lcl|NZ_CP026520.1_prot_WP_042227826.1_2746 [locus_tag=PC41400_RS13980] [protein=GNAT family N-acetyltransferase] [protein_id=WP_042227826.1] [location=3069312..3070289] [gbkey=CDS]
MVITIRNMCIPEDYEGTAVLLSTFYSEPITVQRMREEDAASPQKGELTRDENGRLTGHDRQRLVAVNESGKIIGYGCSWR
APWTAPGTLYQKVLVDPAERKNGIGRGIYAKLDAYADENGASKIVGEVRDNDRDSQAYASGRGFVRDRHQFESLIELDGL
AAETDGRSADTPEGITLTTLAQVPGEEMEKRVYELYRKTHPDIPGSEGEFMWYSQWRKENLERESFDPSLFLLALDGDKP
VGAAELYEMKQTNSVYNLYTCVDGDYRGKGIALALKRKSISLAKERGYGYIRTNNDSQNAPMLAINRKLGYVPVPGYYRM
FKILE
>lcl|NZ_CP026520.1_prot_WP_084706548.1_2747 [locus_tag=PC41400_RS13985] [protein=stalk domain-containing protein] [protein_id=WP_084706548.1] [location=3070468..3072114] [gbkey=CDS]
MKSQPPRMLTIFLSAAIIGSCMPAAAMASQDGKIPQTLASQASAAGEDYLPLRALCEWVGIPIHWDEAKRTVTAAKDKHI
LSFVLAKSGMPAADLEGISLAPDEVILLDSTAYIKRETAVKAFGIEAAWCGGKLEIPARDLAARARAFVTQLQKNNRAGT
DALTNDAFKEAGLPEGTVQLVKQLAQFPITRERIESNAVHRTVILTSESPKMNFDLEIRFDKDGCVDDFYAGMAFEGYQR
PSYDKPDAYKERRMVIGTGDHAVEGTLTVPAGKGPFPLVILLQGDGELDRDSTVFAQKPFRDLAVGLANDQVATLRMDKT
TREHFAQLNSHYTIGDEFVEPVLEAVKAARVDPDVDPGRIYAAGHSRGGWMIPRLLAKDTDSLIKGAVVLAGADPRYTEI
DSYEHEELGGMIPKEQLAYYRKQLELVKADSFDPKSPPAAFELPPNPYWWHDIRGYSPADEAKLRSAAMLVLQGEEDFQV
PLASLKGWKEVYAGNPAVTYKTYPKLTHLFTEGKLENGIGNYKIPASVPETVIADIAAWINGSASVPK
>lcl|NZ_CP026520.1_prot_WP_042227822.1_2748 [locus_tag=PC41400_RS13990] [protein=hypothetical protein] [protein_id=WP_042227822.1] [location=complement(3072176..3072415)] [gbkey=CDS]
MAAYIFRFLKPFVLLGVILGVIVFILNVLGLEIPMVVGTTTYRGTEAAIMELIGIPVALVLLGTIIGSIAYMSNNSQKY
>lcl|NZ_CP026520.1_prot_WP_042227820.1_2749 [locus_tag=PC41400_RS13995] [protein=hypothetical protein] [protein_id=WP_042227820.1] [location=complement(3072513..3074153)] [gbkey=CDS]
MNAWNITYQVALADFLERTRRFSFLVTLGLTILAAYLFVPPSDAGYVTLYFEEFRGIYNSAWVGGTTAVSTSLFLSFFGF
FLVRNGIERDKRTRVGQIIEGSSLKKSHYLLGKALSNFLVLAVIAAVVIPVAICMQWVRGEDYTLHLGAYLAPYLFVVIP
ALAVVSAMAVLFETRPVLRSPFGIVLYLLLFTGVTMSTLYSGMGNDVITKAMREKLLLTHADYEGTYGQGILFLESPLKL
FVWDGISWTWDVVIKQAALLLVAVLICLLASRLFRGFKESPVVVQGPNAGTAAGQTVPARSRGGRAPAPEQAKKTPAEPA
AAKSPALASAAGLTPVRVNNRFLPLVGAEFRLQFAAFPPGWYVVAVILGLLCLFLPFTTALRWIWPLAWIWPISLWSSMG
NREYQYGTLSVLASSPGYVRRQIPAVWLSGFLAVLLTGSGMLIRFIVTGDVVHLIYAVAAAFFVPSFALALGIWTESKRP
FELIYMLLWFLGPFNQLYYADFMGTAVPAGSANHLTSASLLIIGVLLAASVLLVLATYSKRKLKVK
>lcl|NZ_CP026520.1_prot_WP_084706547.1_2750 [locus_tag=PC41400_RS14000] [protein=ABC transporter ATP-binding protein] [protein_id=WP_084706547.1] [location=complement(3074170..3075021)] [gbkey=CDS]
MSNIGKQYQENVWGLRHVDLQVQPGILGLLGPNGAGKSTLMRLLATISEPSEGTITWNGVGIRENPEEIRRILGYLPQDF
GVYPHLSAHEFLNYMAAVKGISGKAAEKRIDELLGLLHLYDVRKRRIGSFSGGMKQRIGIAQALLNDPKLLIIDEPTIGL
DPAERLSFRNLLTDLSGERLIIFSTHIVSDIEATADRIAILSKGRLISDSNPGELLRHVQGKVWNVVIPESQLQEARAKW
LISATVRREDGLLLRVVSETPPNEQATAVHAGIEDAYLYLSAG
>lcl|NZ_CP026520.1_prot_WP_042227818.1_2751 [gene=ectB] [locus_tag=PC41400_RS14005] [protein=diaminobutyrate--2-oxoglutarate transaminase] [protein_id=WP_042227818.1] [location=3075520..3076821] [gbkey=CDS]
MNSFELLESNVRSYSRVFGDVFRTAKNSMMFTEEGKCFLDFFAGAGALNYGHNPDFIKGKLLEYIGSDAVVHGLDMYTTA
KRTFIETFNELILAPKGLDYKLMFCGPTGTNAVEAALKLARKAKKRTGIFSFSGGFHGMSYGALAATGNLYNRGGAGAPL
HNVTFMPYPYGFNESFDTIGYMENVLTDANSGIEKPAAVILETVQAEGGVIVAPAEWLQRLSALCKEHDILLIVDDIQVG
CGRTGSFFSFERAGIRPDLVVLSKSISGYGLPMSLLLLKPELDVWQPGEHNGTFRGNQLAFVGATAALEFRISSNLEESV
KEKEAWVSDYLQKEIVPLHPAITVRGLGLIWGIDLGGLEGGEQAAKFVASRCYDQGLIIERSGRNDTVVKLMPALTIDSE
LLEQGCKIVKEALAAYFEQAGLSARKEATLEPV
>lcl|NZ_CP026520.1_prot_WP_042227816.1_2752 [locus_tag=PC41400_RS14010] [protein=thioesterase domain-containing protein] [protein_id=WP_042227816.1] [location=3076880..3077602] [gbkey=CDS]
MKKIFLFCLPYAGGSSVIYTRWKPLLDSGIELVPLELAGRGRRINESFYENVDEAVNDLLPVVEPYLDKGQAAFFGYSMG
SLIAFELIHRIREKHGQNPLHFFAAARMAPHIQRTEPPLHGMEPELFKTKIADLGGTPPQLFENSELMNLFLPLLRADFR
LVETYAYAEKAEKLDCAMTVLAGKRDTMRETDVLDWRRHTTGAFTAHSFDGGHFFINDFQKEIVDILHASLGIRYSSLYA
>lcl|NZ_CP026520.1_prot_WP_042227814.1_2753 [gene=fabD] [locus_tag=PC41400_RS14015] [protein=ACP S-malonyltransferase] [protein_id=WP_042227814.1] [location=3077617..3078834] [gbkey=CDS]
MVTTAWLFPGQGSQYVGMGAAWYARYDGAKALFEEASDLAGYDMKSLCFEGSAAQLSRTSFTQPALLTVSVLASRVYLHE
IGIEPRYSAGHSLGEYSALVSSGVLTFAEALGLVIQRGKFMEEAAEAGLGSMIAVRTSDQLMLEELCRRHSSDGQPAAIA
GYNSGSQLVVSGHRNAVASAAEELERLGIEVTRLAVSGPFHSVLMQPAAESLEEALNKVTFKEGRWAVISNTTALPYPDA
AAIRRGLVDQMTTPVKWLQTVRFLAEEGVELAVELGPRTVLKNLTRDSSGQIRPYAYDREADAESLADLFPVRQFIEAGF
SAAVSAPNANWNEEEYAQGVLEPVRRVREMLSSYTEMNGSVRDAYGLRQDVQESVRQVLRTKKISEKEQDETIRRLFTDS
FTKSL
>lcl|NZ_CP026520.1_prot_WP_241688865.1_2754 [locus_tag=PC41400_RS14020] [protein=amino acid adenylation domain-containing protein] [protein_id=WP_241688865.1] [location=3078856..3091599] [gbkey=CDS]
MNVEQFLTLIDVIRDRGDKKSSGIIFINGDQNEESITYGELLEDALGVLHHLQSSGMKAGDELLMQIDDNRSFLNVFWGC
LLGGIIPVPVTVGNNDEHKMKLFKIWDTLNNPYLITEGKALDNLEKYALQNGKEEVFEKVREVSLFTDKVHYSTGDKGVI
HEAKASDLAFIQFSSGSTGDPKGVMLTHENLIYNTRDMAVQTEMDETDAYLGWMPLTHDMGLIAFHLTCVMSNTRQLIMP
TALFIRRPSLWMKKASEHKVTQTCSPNFGYKFFLAQYKPENAADWDLSRLRLIYNGAEPISSELCDVFMEAMSKHHLRST
AMFTVYGMAEASVGMSLPPLEENFRRLHLDREHLNIGEKVVQIDASDNRCLTFVDVGHPLQNSYIRICDDANLPLDDYVI
GNIQIRGMNVTSGYYNNPEATAKVITPDGWLITGDLGFMREGRLIITGRAKDIIFANGQNVYPHDIERVAEEVEGVELGK
VAVCGAYDPATRQDRIIVFVMHTKKIEDFAPLAKRLKGHLNYRGGWDIQAVLPIRRIPKTTSGKVQRYKLAQDYTDGKFE
EMIAGLQKIWEQEKQSARAAVIPRSVAEQKLLAIVKQILQVDDIDTHESYFDIGASSLQLVQIVEELETQLGVQVSVTDF
FSYPTISKLAGFISQGDEADRVEQSDRLADSGQETDGDIAIIGMSGKFPQADNLKQFWTNISEGRDCIAPYGEKRRKDAE
AFISHLNTEGRDLSLAEGGYLDEIDTFDYSFFKLTPREASLMDPNQRLFLQTVWSAIEDAGYGGSNLSGKRVGVYVGFSK
TSFEYERLLSEVSPASLPNFAIGNLSSIISSRISYLLDLKGPAVTVDTACSSSLVAVHLACKAILSGDCDMALAGGVKTI
LLPVKAGIGMESSDDRARAFDDSSDGTGWGEGVGAVFLKPLGKAIKDKDNILAVIKGSAINQDGSTVGISAPNASSQADV
ISQAWRNARIDPETVTYIEAHGTGTKLGDPVEIDGIGKAFRRRTKRKQFCAISTVKSNIGHLYEAAGIAGLIKTVLSLQH
KQIAPSVHYRVPNRKISFEESPVYVNTQLKEWKTDGAPRRCGISSFGFSGTNCHVVLEEYTGAPEPLPEASAANSGPEGS
VPLLLTLSAKTEEGLQELIRDYDQLFARTDERDMQNVCYTAGTGRSHLTYRLAVTASDAAELKQKLQRLHREGKGSDGVY
IGKIDSAQVQEYTHPGNFADLGAEAVQELARQYVQGASVDWKALYAKGSFRKVSLPTYPFQRKRCWVDVPVKAGQAASAA
QSLHSAPTGQPEREQYISAKEQVSVVNEQLQAETASARHADSIRQTLITMIGNVSQLSTEELQPQTHFLELGLDSINLSQ
VRHSIKDAFGLDVPMSDFFESLTNLDLLTAYIVERVPQAETASVIRSEEVLQVSPAETASAVYERTVPAEPYSYSEPAGT
AGGQGQAEAASVPVVGIAGSAASSAVERIVEQQLQLMSQQLQVLGNRPAAQAPLPVQPAAVPGPAPAVRVIAASPAQSDA
RLHAGTTREVAAAVETRPAPPQVNGEEPKPYVPYKALDLKARESLSQRQEKHVRELIERYTARTGRTKAYTQQYRDVYAN
NRNVAGFRPLLKEMVYQIISERAEGSKLWDLDGNEYVDLTMGFGVNLFGHNPAFVREKIERELANGMCVGPMSNLAGQVA
ESICRMTRVERIALYNSGTEAVMVALRLARAATGRAKVVIFAGSYHGTFDGVLALGSAGTNKEHSTPLAPGILQNMVDDV
VVLNYGSEDSLEYIRTHAHELAAVVVEPVQSRRPDFQPKAFLEEIRRITAQSRTAFIFDEVITGFRLHPGGAQAWFGIKA
DLVTYGKVIGGGMPIGIVAGSSAFMDGIDGGTWKFGDDSFPQNEHRRTFVAGTFCHHPLAMAASLAVLNHLEASGGQVQA
ALNARTARFADELNTYFGREGVPIKVVHFGSLFRFVLKGDLELFYYHMLDKGIYIWEGRNCFLSTAHSEEDISRIVQAVK
DSIGELRKGGFLPDRDPSPDDDDPSGSRLPADKLPADGQGETFPLTPDQKQLWFASVSNRKDSPALHETALLRFRGPLRN
EVFEQAVRAVTARHEALRLVISQDGETQIVTDALNVPVPVVDLTGFNSAEQERQIRIWLHSESQTPFDLEQKTPLFRIHL
LRTSADEHTAVFTFHHLIADGWSMALFVQELERIYSALVRQEAYRLPEPAPFRRYAAWQSEQLKKPQSAEAAAYWNGLFE
KQVPVVELPSPVRGTVLPTSRGGRHTTILQTSLVQELKKTSIKLGSSLFVTLLSGFHLFLHRLTGQSDITVGVPTAGQSQ
MEAPFLIGNCVNLLPVLSRASSEQSFAAYAGSIKQRMNELDAYQAYSFAALAERGLRNLPVLTAVFNMDRPLPKLAFHEL
DTELLVHDPAYCKYELFLNVTEISKELRLDFDYNADLFQPETVETWSRYFIRLLESIAQGGDKRLPELSLLSAEESERVW
ARWALHADENGSYECILDAYKYPAPPGTIGEVYAVSGGKVKSPSNRFTGTGKWAYSYPSGAIKRLGKSSRSVFVRGNRVF
LDQIEEHLLHTLHLEACVVTAQLHPENEAVDYMTAYIVANPSKIWKEIESFSLLHETLPDYAVPRYVIPMAELPLLPDGE
TDFAALPAPSGGNLPLPGKQNLQGSVTDKLTAIWKDVLGAAHVNATDNFFQLGGDSLKATVILSRVNKEFGIHIPLSELF
ALQTVGDLEQFITGGSGQTYEPIQPVQPKQSYPVSSAQKRMYVLDQLGGGGTAYNVPGRLFIEGEIDRERLIRALRQAAA
RHESFRTYFELENGEVVQKIAEEADLDVPVSTVRAEEIETLQMQFVRPFDLTKAPLFRAGLFELETGGSVLLIDVHHAVA
DGYSMAVLLDDFIQLYQGRLLPPRNVHYKDYAIWQNEQLAEGKFSRQEAYWLQTLEGELPVLQMPTDYPRPAAQSFKGDS
FVQLPDGKLTKALYSLAKNNETTLYMVLLAAYNVLLAKYSGQEDILVGTPAAGRRHADTENMAGMFVNTLAIRSKPESSL
AFGQFLQEVKRQSLEAFENQEYPFEEIVEKLQLVRDLSRNPLFDTIFSLQNVGTDVVKTGKLSFKPGEFNVGISKVDFSL
HVTEQKDGLLFTWEYATDLFARETVERLARHYSVILETIAGSADILLGDIGMLTATEKEDLISGLNATAAEFPTGITIHE
AFEEQAAKTPEAAALVFEGRSLTYRELNERANSAARTLRELGVGPDRLVGLMVPRSIEMIIGQLAVLKAGGAYVPIDPDY
PSARVLYMLQDSEAAVLLTSRSLIEDLDYDGSVLFLEEEEIYRNDGSNLDVGGTKPEHLAYVIYTSGSTGNPKGVMLPHR
SVMNFVTGMKEQIAFRPDKTILSLTTISFDIFVLESLVPLLTGMTVVVGDRHHQLEAEALGELIRSREIEMLQMTPSRLQ
MLLANEEGKRALNGVKEIMVGGEALPHALLEHLQTLDGLRIYNMYGPTETTVWSAVRELTQASRIDIGRPIANTQVYIVD
ARSNLQPVGVPGELCIAGEGLARGYWKREDLTAEKFVPNPFVPGTKMYRTGDLARWKPDGNLEFLGRIDHQVKVRGFRIE
LGEIENALIRVPSVQEAAVTVNQSGGHEAVLCAYVVPKSDSAISAAEVREQLARQLPYYSVPSHYVFLDKLPLTPNGKTD
RKSLPEPDDLFRPDTEYAPPRNEVEEALAGVWQDVLGRTGIGIHDNFFLLGGDSIKGIQVVARMRAEGYALDIKQLFEHP
SIGDLGAYVRSSVREIAQGPVEGEMVLTPVQNRFFEQGGYESASSLSALLFSEAGLEENHVRSAWDKVLKQHDALRLRAV
LKEGNLKLEHRSLEEDAGLSVIRLDGDKSDFRAAVELQSKGIESGLSLKEGPLAKSVLFKTGEGDYLLIAIHRLITDETS
FSVILEDLATVYGQLIAGESVSLPLKTDAFRTWAAHLAKYAESSQSQAQREYWTLPGNAEEWTVPPDKETEAGLPPLRNS
VCAVIPAEKTSQLLTGVKEIYSMEAREAILSALGKTFGESSGLKRFPVELTADGRKLCADKRLDFSRTVGWFDTQYPLVL
DTSHTGDLAGYLKQVKETVRQVPDGGIGYGALYDLASSGRQSEETAVRKKPRIGFVHLEPSGVVTPYFDSFNAGDRITDE
LWAAREYSWYLKSTFESGVLTLRLDYDGHAYTEKTASDLLHRLEENLDEVLRHCEAKTERELTPADVGANDLDLDEFEDI
KQFYESV
>lcl|NZ_CP026520.1_prot_WP_042227813.1_2755 [locus_tag=PC41400_RS14025] [protein=non-ribosomal peptide synthetase] [protein_id=WP_042227813.1] [location=3091623..3099581] [gbkey=CDS]
MSKGLEIEKMSHLTPLQEGMLFHALMNPDSHAYLEQVSLSIRGDLDMQALERSFRLLVQRHETLRSNMYHKNVAKPRQIV
FKEREASVFFKDLALLQDSERNAELESLIQADKNRPFDISKELLVRLTVLRTSQQEYTLLFTFHHIIMDGWCIGIVLEEF
LSYYEAEKNGRQASLPEVRPYSLYVQWLQKQDQTAASDYWQALLEGVEEPSGLPGNTAISGGKGYLPSERKFTVGADTGR
QLRGIAERSGATISSLFLAAWALILGRYNRSEDVVFGTVVSGRPPEIEGIERMVGLFINTVPVRISLQSDNTFTGLLKEI
QRQTVQSRAYDYYSLAEIQAQSALKQDLIGHIVVFENYPLQELIGQEDLENRLGFAITEAGLSEQTPYDFNIEVEDGDVL
NITLSYNGHTYGEAEIARIQGHLETVLGQASSQPEKIVSEFTLLSEQERQELTEVWSGTSAAFPEGTFHELFEAQAAQTP
DKTAVVYEETSLTYRELNERANVLARILRKQGVRPDSPAAVLLERSADMVTAVLAIQKAGGAYIPIDPTYPAERIAYMLR
DSGAGVLLTQAKLTGLLQGADYPGSILDMQGLLEAAENQGFVKPLKAGEQAASNSGEEKAWLSGLNPEEAVSNLEPVNKP
DDLAYVIYTSGTTGNAKGVMIEHRHYVNTSFGYRYGYRLREFDVKLLQIASFSFDVFAGDLARTFVNGGTMVICPQEVRI
DPSALASYLEKHRITVFESTPALILPLMEHVYDHGTDISALKLLITSSDSCSVHDYRTLLSRFAGSMRIMNSYGVTEAAI
DSSFYEEAADKLPESGNVPIGRPLGNHRFYIVDSALRPVPAGVAGELCIGGDSVARGYLNRPELTAEKFVPSPFAEGGRL
YRTGDLARWMPDGNVDFIGRIDYQVKIRGYRIEPGEIESALLKLEGVAQAVVTDRTDSSGHKYLCAYVAGPADEAWLRKR
LAAELPGYMVPAHIVRLDSLPLTPNGKIDRKALPAPTGDPAPVSAYAAPRTLTEAQLAELWQEVLGVKTVGIDDSFFELG
GHSLKAVLLIGRIAKQFSVEVPLREVFAHPTVRGLAPLIAGSVESAFADIEPVGEKDYHPVSPAQKRLLIIDRLGDAESA
YNMCAALEMTGPLDRERLEEAFRTMIDRHESLRTSFAVIDGAPVQHIHKTLDFRVSYRELDSDGHHEARNEAAAASVSGL
PAEAESGSSGAQVGAKQPDSALESLIMPFIRPFNLEQAPLIRAELIRLEEGRHLLLADMHHIITDGISIEVFVEEFSRLY
AGETLPPLRIQYKDFAAWQNGQIAQGAAEKQAEYWKELFGGGEDVPVLDMPADYIRPAIKSFEGDSLTYEIGSGLTADLQ
RLCAQTGTTLYMVLLAAFNTLLHRYTGQEDIVVGSPVAGRPHPDLETMLGMFVNTLPMRNFPSADKPFSHFLKEVKDRAL
HAYENQDIPFEEVVERLQLRRDVSRNPLFDVMFVLQNTGMNQFQFDEITFRPVEFEMGIAKFDLTLSVEEFSDRLGLSFE
YAVKLFKRETIEKLSRHFLNILETVVRNPAVNLSDIELLTETEKHRIIVEFNDTKALFPEDKTIHGWIEEQAAEDPDRIA
VVFGANRLTYGELDERANRLARTLRAKGARTDGVVAILTERSAEMMIGLLAILKAGSAYLPVDPAHPRDRVAFMLDDSGA
RIVLTASKWAEQLPAGYECVLLDEEKSYHEDFSPLSSEGSAGPGNLAYVIYTSGTTGNPKGVMIEHRALVNRIHWMQKKY
PIGRDDVILQKTPYSFDVSVWELFWWGTQGARVVFLAPGGEKEPQLIVDEIKRSRVTTLHFVPSMLSVFLDHIEQNRQAG
ELGSLRYVFASGEALQAQHANRFDRLIGTPGGAVLANLYGPTEVTIDVTYFDCPQGQQLEQVPIGKPIDNTELFIVDAKH
QIQPVGVPGELCIAGTGLARGYLNRPELTAEKFVPIPHRPEAKMYKTGDLARWLPDGSVEYIGRIDHQVKIRGYRIELGE
IEAKLLNRDEVKETIVMDRTDANGQKYLCAYLVCAEDAELTVSELRSGLAERLPDYMIPAQFVFLDAMPLTSNGKINRRA
LPEPDGSVRLTADYVQPETEMEHTLAEIWQDVLGTKKVGVRDNFFELGGDSIKALQIAARLNGRGLKMELRDLFRHPQIE
LVIPYIQVVQRTIPQGTVQGQTELAPIQRRFFERHHTDRHHYNQAVMLESKARLNEAVLRQAWDKLTLHHDALRTVFRST
EEGGVTAETLDWKEGETYELLAYDLTPAREPGPEIGTLADKLQSGFALNRGPLVKLALFRTAESDHLLIVIHHLVVDGVS
WRILLEDLETAYGQALDGHAVSLPAKTDSYREWAARLAEYANSSRAQSELSYWSAVSSQIPDSLPYDNKAASDKAEDGKD
FEIELDRESTEKLLKHAQQAYNTEINDLLLTALGMTFQEWSGLSRIAVSIEGHGREEIMKDVDVSRTVGWFTVMYPFVLD
MNGQGDLGHQIKLVKDGLRRVPNKGIGYGILRYLAEGVNPEELPFAFEEPQISFNYLGQFDGEGDGGARFTPSSHQTGDE
IGGGSERPFAFDINGQVSDGVFRLTFNYNRHQYRDDTVEKLAGGFKKYLLELIRHCCEKETVEQSPTDFTYQNLSIGQFE
NMAAQLAGKLKL
>lcl|NZ_CP026520.1_prot_PC41400_RS14030_2756 [locus_tag=PC41400_RS14030] [protein=amino acid adenylation domain-containing protein] [pseudo=true] [partial=3'] [location=3099745..>3105498] [gbkey=CDS]
MLDRDSEAYFQQNIITVGEVLDRELVQQSLQLLIERYDIFRTVFVYKETERPLQVVLQEREAPSVHTEDVSGLSEDRKTA
ALEQFTGSDRTNGFDLEKDVPLRLALFKWSETTCKLVWSFPHIVMDGWCLGIVANDFFSLYTSLRDGIRASLEPVYPFSS
FIQWLENQNREEAADYWSKYLDGLENETTVPGFNRAEGSTPFRLEHHSLKLDAALTQKMEQIAKRQQVTVSTVFQTAWGV
LLQGYNNSTDVVFGTVVSGRPEEIKGIENMVGLFINTIPLRISGEKSESFAALLSRVQSDWMESNAHSYLSLADIQAQSQ
LKHNLIRHLVVFENYPISEELIGTGDETGGGRRADGEPAKNSLRITEVDGFEQTNYELTLTVIPGRELEIKFSFNAENVE
FGGIRAIAGHLHQVLEQITETPDIPLDSINLLTDEERSQLRSFNDTGASYPAEKTMHGWFELQAAMTPEKIAVIFGEKRL
TYRELNVRANRLARQLRLKGVEPDSVVGILCERSIELMVGLLAILKAGGAYLPIDPGHPQERIDYMFADSGAVLVLAQKE
LLERAPAGAETVELDAENLLTGDGSNLSRGSGPEHLAYVIYTSGTTGNPKGVMIEHRALINRLHWMQKRYPLDEKDVILQ
KTPYSFDVSVWELFWWGAYGAKVVFLEPGAEKDPAQIAKAIDKYNVTTMHFVPSMLSVFLEHAESRLEPGALASLRYVFA
SGEALNPQQANRFNSLFYQTNGSRLINLYGPTEATIDVSYFDCSFGKDSLDRVPIGKPIDNIRLYITDNAGRLQPVGVAG
ELCIAGTGLARGYLNRPELTAEKFAAANFDPSGRIYKTGDLARWLPDGNIEYLGRLDHQVKIRGYRIELGEIEAALLRHP
MVKEALVMDRTDGQGQKVLCAYLVCGPDPAPETVLLRESLAVKLPGYMIPSYFVFLPEMPLTSNGKVNRRALPSPDTAWD
ESGYIPPEGETEEALAAIWKEVLGVEHLGAGDSFFDRGGHSLKAALLVSKIHQRLQVDLPLRQVFESPTIRTMAEAIRKE
AYSAHQAIEPAEPQDVYPVTSAQKRLMVLDQLEQESTAYHIPHVLISRVPLDRRRLQTVFEKLIARHEPLRTSFEQIESD
IVQRSHPEVVFEVEEVTLSGAEAPSLTELGERAEIKEKSGIEEIPAFSGFSAVLKTDEEITPENTDGNGASGHPEERTAV
QDILTTAEKEALDRVIEGFIRPFTLGEAPLIRVGAADLICGGTALLIDLHHIIADGVSMAVIANEFNRLYAGEELAPLKI
QYKDYAVWQQKRSSSGVLDAQEAYWTEKLSGELPVLNFPTDYMRPPLQSFEGDRYVFTLGSSEANALKQAASRAGATLYM
VLLASFNTLLHRYTGQEDIIVGSPVAGRTHADLQPLIGMFVNTLPMRSFPAGDKTFSQLLKEVKRTALEGFQNQEIPLET
ILDKLSLKRDLSRNPLFDMMFIFQNLGIEEIGQPGELFQSYEFGHAVSKLDLTLQAEENETGGIDFHFEYATGLFRKETV
QRLAGHYVRILEAAVRNPDIKLSAINMLEERERRMLVESFNDTVVAFAHDKTIHRQFEEQAALTPDAPALLFGDQVFTYR
ELNEKANSLARTLQSHGVGPDRLVGLLTERSPSMMIGLLAVLKAGGAYVPIDPDFPAARVAYMLEDSEARVLLTEGALTE
GLQTAGAVISLDDEGVYSGETGNLEPASEPHHLAYVIYTSGSTGNPKGVMLQHRSVSNFFTGMREKIEFVPGRKILALTT
VSFDIFVLETLLPLTCGMSVVLAETLHQTEPGALSGLISAHGMDTLQITPSRLQMLLGSPEGASSLQNVRNILVGGEAMP
VALLRSLQEFPGLNIFNMYGPTETTVWSAVQNLTRADSIDIGRPIANTQIYIVDAGGKLQPAGVPGELCIAGEGLARG
>lcl|NZ_CP026520.1_prot_WP_275938053.1_2757 [locus_tag=PC41400_RS30580] [protein=phosphopantetheine-binding protein] [protein_id=WP_275938053.1] [location=3105567..3106244] [gbkey=CDS]
MYKTGDLARWLPDGSLEFLGRLDHQVKVRGYRIELGEIETALVRHEAVEEAVVLAREDASGGHTLTGYLVVKTEIPASEI
RTFLGQILPDYMIPAAFAQLEKFPLTPNGKTDRKALLGMDAVQTSVAVYVAPGDEIEEQLATMWAGILSAARVGVHDNFF
ELGGHSLKASLFVAQVREQWGVELELRQLFRQPTIAEIAEHLRQELDEIRRLEEILREIEALEQV
>lcl|NZ_CP026520.1_prot_WP_241688866.1_2758 [locus_tag=PC41400_RS14035] [protein=amino acid adenylation domain-containing protein] [protein_id=WP_241688866.1] [location=3106286..3114385] [gbkey=CDS]
MDLSKRIASLSPEKRKLLEARLAQEGLSASRPAGGGAGTSAGLRPLEKMDAYPVSAGQKRLLLIQEIGDTAAAYNMPFVL
LARGPVHVGKIQGAFEALISRHETLRTSFRKRDGEYDQIIHGPDEILFKLDYSEETVSSVEEERNVIERCVSSFIRPFDV
SRAPLFRAALIRLAPERHLLMTDMHHTVSDGLSLQILAEEFNRLYRGEILPELPVHYKEYAYWQQERMRDGVFLKQEKYW
LDVFEGEIPVLNMPADFPRPQLQSFEGGRCEFETDRGLADKLRRLASETGSTLYMVLLAAYYVLLHKYTDQEDIVIGTPV
GGRTHTQLQPLIGMFVNTLAIRQAPAGSKTFLSFLNEIKEGTLGAFDHEEYPLESLLEKLGLRRDVSRNPLFDTVFSLQT
KGSQSMEMEGLLLEPYEAQSGVSKFDLSLLAEEDEELTFTFEYATKLYAKQTVERLAGHYIQILREIAENRSLMLSQIGM
LSAEERRLLLEELNETAAPYSRTSTIHGLFEEQVEKHPDRIAVSFEDRRYTYYELNARSNRLAAAMQDLGVGPDTIVAIM
PERSPEMVIGILAILKAGGAYLPVDPEYPQERISYMLEDSGTRILLTHRHLAGSTEFTGHTVFMEDEQWYTGVETNPHVS
VTSDHLAYVIYTSGTTGKPKGVMIEHRGMANMKLFYENTLGLGPEDRIVQFASFSFDASVWEMYMALFMGGGLYLAPKEK
LVQLDSFEAFVRDNGITLAMLPPTFAVYLDPGRLPSLRILITGGSESSPELVRRWSPHVDYFNAYGPTEATICATVWEAK
RGTAVQNHRMIPIGKPLYNTQVYLVNADLQLVPRGTAGELCIGGEALSRGYLHREALTGEKFVPNPFDPGRKMYRTGDLA
RWLPDGNLEYLGRIDHQVKIRGYRIELGEIETRLLQHGGVRETIVLVHEISSEKTLCAYYTSEGESPAASELREHMASGL
PSYMVPAHFIPLKEFPLTPNGKIDRKALQAPDPSSGASDEYTAPETETEVRLVSLWQEVLGAERVSLHDNFFERGGHSLK
AALLLAKISKEFGVELPLRKVFEAQTLQAQADAITAKGRTEAAYAPISKATEKPWYPVSPAQHRLLVVQEISGAATAYNV
PLILEVKGALNPEKVNETFRKLIARHEALRTSFTRIQEEYVQVIHDPAQISFQVEVTELADLSTDWTDDQNEAHIKTIAD
GFIRPFDLGTAPLLRASLVKTEVERHLLLIDMHHLITDGTSLERMTEEFTRLYRGEELEELSVQYKDYTLWQNQLSADGT
VARQEKFWLDTFSGEIPVLNFPLDYARPQLQSFEGAVYGFEADHDLTRRLQQAASRQGATLYMILLAAYNALLAQYSGQD
DLIIGTPVAGRPHADLEDVLGMFVNTLPLRNYPSGSKTFAAFLDEVKQNALQAFENQLVPLEDLILKLQLRRDVSRNPLF
DTVFVLQNTESREPGADGVEFAAVEHHNGISKFDLTLTAQETESTEGPLLSLSFEYATKLYKEDTIRKFADHYVQLLKQI
SQNPDMKLSDIDLLTVSERDLILEQFNATKTEYPRDAAVHELFEEQAALRPDEAAVLFGDKEITYAELNRKANQLAWKLH
EIGVAKDTLVGIMAERSPEMIVAILAILKAGGAFLPIDPSYPEDRIAYMLDDSSAAVMLAGDPNSAQSLLGSFKGTVLSL
ADPSVYTGSESNLPRTANAESLAYVMYTSGSTGRPKGVMIEHRSVVRLVKNTNFVTFGPSDRILLTGALVFDACTFEIWG
ALLNGLQLCIVPESVFLHAGKLEEALQAYGITTMWLTSPLFNQLALTNPALFAPLRYLLVGGDVLSPTMIASVRNSCPQL
TVINGYGPTENTTFSCCFPIDREYGNIPIGRPIANSTAYVVDLSGRLLPVGLQGEICVGGDGLARGYLNREDLNREKFVP
NPFRPGERMYRTGDLGRWLPDGSIEYLGRMDQQVKIRGYRIEPGEIETRLLGHESVREAFVAVKESGEGLKFLCAYIGTS
DFLTPSEVKAYLSGRIPDYMIPARFVFLPSLPLTKNGKIDRRALPEPEQDGLSSSPYAPPRSAKERLLAGVWESVLGVER
VGIDDNFFELGGHSLKAIQLISKAQELGLEIGMHQLFQVQTIRGLSELLPAFKEAEEAGSESPGKHSDHVRDNAVTKIFE
EAEQLLRDTFGPGFGLKADMREDKKVVILQLFSDHEKRLPEITQFIRSNFHKDLHPHFIEPQLVSEEGDEEKVAAAEIGQ
ALQAEALNLTESDLQDTLMRVLKKVSSAHEAWGTALTQMPAAEQYSLAPIQEYHLAYPEASGTILSFDEIIDTQRLNAAL
QRVIEKHDLLRSQLVQTGEQWGWELKEKSPGFALPVVDLSAYLPSVQQRIMSGLLKELYLRPYELGGSAALYRFALISLN
LREHILLLPCSHIIFDGMSSEILRSDVLAYYNADEDQSSRHEGNRPDHSYRSYVGQVRQGPAGIGDAEINERFRLESFYQ
QTSVIGEALQGKKRSKATKYRHEINMEGQDAGTDLAERMWEKALDLAERFFSRYVGLEEVPVWLTNYGRHYTGRHYYDTI
GEFIDQIPVLLGGISGDEKRSVQNLLQAASTHNLNFFNLIYNRETAEDYPLSGYYLQNSLHKLPIVVNYLGETQQDGELL
QELDDPDIDSTEVRPVIYFTVQHTGNLLHISLLLPYEEDEETIRRMFESVIPTFQTIKL
>lcl|NZ_CP026520.1_prot_WP_042227809.1_2759 [locus_tag=PC41400_RS14040] [protein=MFS transporter] [protein_id=WP_042227809.1] [location=3114400..3115683] [gbkey=CDS]
MSLFKNSLFVRFYLSRTSANIADSIYSIVLLYFIQKSTASVSFTSFTFTAISAASIFSFLLGPLVDRYSPSLLASISLFV
QAVLILAVPFLITDGGANLLAILILVFVASCFSMLFYPANNKMLPQLIQSPDRIVKANAMISSTDQLINVAGYLVGASII
LLLGMKNTFYLASGMLFLSGLIYVQLKKQMEPEKSEGGAPKGPLKIREYTSELVEGYRFVAGNPFLRIMLPFFALTNFSM
SILIITVPSMSVSYGSPIYYSLLYIAFFVGIFVGSALINVLKKNGLMIAISWLLMGASLYLFAIVPALWMKLSAILLMGV
FTGIINVLQTSLIQIITPSQLMGRVMAFLHTLSNAALPLGAFIGGILALRFELDSVLFISALVIVACGVLLLALKAVRTF
QIPSETADKEKTEDNLAGKGAVAKEFL
>lcl|NZ_CP026520.1_prot_WP_042227808.1_2760 [locus_tag=PC41400_RS14045] [protein=HD domain-containing protein] [protein_id=WP_042227808.1] [location=3115856..3116848] [gbkey=CDS]
MFISDPIYGEFIVDGLLEELILSSSVQRLKGIYQGGASYLVNEKWDVTRYEHSVGVMHLIRRLGGSLEEQVAGLLHDVSH
TAFSHVIDYVLDCKEEDYHEKIYGQIITESEIPKIMEKYGYAYKSLLLDDSKWTLLEQSAPALCADRVDYTLRDQYHDGK
ITLEEINLFLEKLTSFDGKMMLTCIESAEWFVKTYYREVVDFFMHPLNAYGCYFLAETVKVALAKKTITLGNLLLTDREV
LDLLRASDDPDTQRLLKHLHCGVQVKESEIDYDFHRTNKMRLIDPDVMYGNKIWKVSELSGSVRRWGEEAKRKSERGVYV
KIVSAELPIG
>lcl|NZ_CP026520.1_prot_WP_042227806.1_2761 [gene=clpP] [locus_tag=PC41400_RS14050] [protein=ATP-dependent Clp endopeptidase proteolytic subunit ClpP] [protein_id=WP_042227806.1] [location=complement(3116945..3117532)] [gbkey=CDS]
MSNYVPYVVEQTSRGERSYDIYSRLLKDRIVFLGAAIDDQVANSIIAQLLFLEAEDPQKEISLYINSPGGSTSAGFAIYD
AMQLVKPPVHTICTGFAASFAAILLLAGEKGKRFALPNSEIMIHQPHGGVQGQASDIAISAKRILGIRERLNRITSERTG
QPIDKVDRDMDRDYFMSAEEAKEYGIIDQVVTTIS
>lcl|NZ_CP026520.1_prot_WP_042227804.1_2762 [locus_tag=PC41400_RS14055] [protein=RNA polymerase sigma factor] [protein_id=WP_042227804.1] [location=3117857..3118585] [gbkey=CDS]
MSTADFLLKGKVKDSTRINQLQQTLKRYCLSMTKSSWEAEDLAQETWLRAMSSQGKGHANVEAMLLRIAKNTWIDQSRRK
QVLARIMKSEQTQAVTRRDEGLLVIEEAFRALLKHLPPLQRTVFLLRDVLGFSSLEAAVRLGTTEGAVKAALYRARQSLE
AVRRDIETGTQVEPAENGMKAYLRMMAAAYEIGDIALLVELAQRDEVEPGVAFGLAQNHILRSGRRRTGAVPHLPSSLRM
TA
>lcl|NZ_CP026520.1_prot_WP_009672538.1_2763 [locus_tag=PC41400_RS14060] [protein=hypothetical protein] [protein_id=WP_009672538.1] [location=3118710..3118988] [gbkey=CDS]
MAEQINEQELIAYIAGKTKADPQKIKSVLKYEQDFINKAHKNAKGEVDIDSDDIADYVLSRPDIKLDELTVDNILDAEMD
FLMEKGIASYLD
>lcl|NZ_CP026520.1_prot_WP_042227801.1_2764 [locus_tag=PC41400_RS14065] [protein=ABC-F family ATP-binding cassette domain-containing protein] [protein_id=WP_042227801.1] [location=3119371..3121341] [gbkey=CDS]
MILVNGQQIKKYHAANLVLDGATFEVHAGEKIGLIGRNGSGKSTLLQLISRQQSIDEGMLTVKKELMIGYLPQIPVEFEG
LTVYEVLSYGCRHLTACQQEMTALEHDMAEPGASADSARMEGLLNAYASVQERFERGGGYEMDAVIDSVAIGLQISKETY
ARQFSTFSGGEKTRIALASQLIMRPDLLLLDEPTNHLDLRGIEWLEQFLQGFAGACVLVSHDRYFLDRTASKMIELEDGE
AFTYQTNYSGYMKEKEERLLQQFAQYQEQQKAIKKMKETIRKLEEWGRIGDNEKFFKRAASMRKALERMQVMKRPILEQK
SADFALNLQDRSAKRVAEFSGVHKSYGERRILKGAEGSLHYGEKIVLVGDNGTGKTTLFKLLLGQEVPDYGALELGSRVE
IGYMAQQEESRDPQKTVLDYFRLEAGVEEGEARGILAKYLFYGTDVFKPVSMLSGGERSRLRLALLVRRKPNLLLLDEPT
NHLDIASREALEEALEEYPGTVLAISHDRYFINRLAQRVWELQDGQITSYIGQFDEYRAKKRELEQTAVDRGHGRPPADV
PGAASTAVRAGEQPLCAAEANAAAKAAAAEARATERKRLQLEAQIAALEAQLAASDAELQQLGGCGDTAQLEARWREREE
LQARLDAEMTAWLELG
>lcl|NZ_CP026520.1_prot_WP_042227799.1_2765 [locus_tag=PC41400_RS14070] [protein=sensor domain-containing diguanylate cyclase] [protein_id=WP_042227799.1] [location=complement(3121451..3122998)] [gbkey=CDS]
MKTKRLRKGLTLRLILSLLVLATVFLTAAVSGYAAYQANLKSLSASYLVNNSQYAQKLSSNTGNLLNLMQTSMNSIAQLA
GKQSFTEKDLDMWFTFNRQYFNSVFVADDEREIKVTLPADAGVVAGTRLLSEASKRAVESKTPFISEPYVSTTGRLIVLV
SSPIFNGDGSYAGFVGGTIYLEEENALSKLLKEHFYGNGSYVYVVDKESRLIFHPNKERIYQTISSNEVIQKALAGKNGS
QKIVNSEGNEFFAGYAFEPISGWGIVSQTPTRVLEEPLEQMISNLLMQALPLLGVVLTVAWWLASWITRPLYTLARYSEE
SAVHGNLTVPASPSIHSGLYEVRQLYQSINSNLNRLKAESLTDGLTGLANRKAFDLLIAEWAERKVPFTLILMDVDHFKK
VNDLHGHLIGDEVLKYLALTIDAFSREEDLCFRYGGEEFGILLKGCDLGRATRIAERLREKVERSKSPIAQPLTVSIGVS
FSYSRDTDVKTIIEMADNALYQSKTDGRNRITIYS
>lcl|NZ_CP026520.1_prot_WP_042227797.1_2766 [locus_tag=PC41400_RS14075] [protein=EAL domain-containing protein] [protein_id=WP_042227797.1] [location=complement(3122995..3123648)] [gbkey=CDS]
MEIPIEKGQEELPFYRTLITVLTEHTPVGIYAMENGALTYANDSYAKPVGYAAKKIRPAEFGFLEPGDCPMTVPLFSEPD
DRRKGAPLRIRTRHKDGHWPFFAGHVLSIMKEDGNGISFDDFDTGYTSLNYLRECRFDKVKIDRPFIEDINRGLNGKQIT
SAIISLAHSFHMNMVAEDIENEIQHKYLLGENCNDGQGYYFSRPLPADSLAGLLKRP
>lcl|NZ_CP026520.1_prot_WP_241688867.1_2767 [locus_tag=PC41400_RS14080] [protein=histidine kinase] [protein_id=WP_241688867.1] [location=3123842..3125242] [gbkey=CDS]
MKRTGRLAGAGFTTVVGIMFFLYYYFVIEEAWEMQAAGAVVCTVIAYFLGGFYDRARYYAAELNLINEELHISREQLRRI
DERYSALQASLDNFSHDLFGIMKIAELEARLLAEVKLIFQIDCVSLIETQAGHPSCSVKTGTGKVPRAVLNKIRQHQGQQ
LSIGEVFEYEEGSFLKIGEFKGRTFWLIFGELTDSMRLPVKRIWLKTLIRYVNVVYENLIVIEDLSAELKRTMEEQETPP
WLLRLFFSLSEHERKRLSQDLHDAALQEQIIWYRKLEQLRTSGLGPDELKEGLAQIGDGLLDVMYQIRLTCNELRPPFLK
EWGLVQALEALFDHIQLHADFTIYFEAEHFRDYLNDEQRIAVYRITQELMSNAMKHSKADEVHVILTGGKQNLLLSYRDN
GVGTKTREQADNGFGSMGIYGMQQRVRSLQGEIQCVSDKSGLKMQVSIPLKPVLHVIPPHGTFVSV
>lcl|NZ_CP026520.1_prot_WP_042227792.1_2768 [locus_tag=PC41400_RS14085] [protein=serine hydrolase domain-containing protein] [protein_id=WP_042227792.1] [location=complement(3125573..3126934)] [gbkey=CDS]
MQNKQKQLQELFAAMSEKGVFNGVFLAADKGEIVFEGAYGLADRETGRALTVHSVFELASVSKTFTAMGILILAEQGKLS
LSDFVEQWIPDFPYKGITVKDLLRHTSGLPDYTPLFIKDWDRSKIATNWDVLEYFNVRRPPLQFQPGENWEYTNTGYVLL
ALLIEKISGLSFAEYMSETLFKPLGMHSTRIHNRRLSGNVPPDFAFGYVYSFETHSYHLPDEQPATDFVVYLDGIQGDGT
VNSTLHDLLLWDQALYTEKLVSKKTLDEAFTSGKLPDNKPYSYGYGWIVEQKEHLGKVVSHSGGWPGYSTNLARYTDHGK
TAIYLSNTDRQLDIQQQIINEAENILFGLPHSVPDVPAELRSVKIDKTVYMLYRGVYKMDMGLDLSVTTGPEYDQLFIQL
SGQPQYEILPLSEERFFVRELQIEIEFAAKKAGKPAHELTLFQYGEHRGVRVT
>lcl|NZ_CP026520.1_prot_WP_009674103.1_2769 [locus_tag=PC41400_RS14090] [protein=MarR family transcriptional regulator] [protein_id=WP_009674103.1] [location=complement(3127022..3127465)] [gbkey=CDS]
MDERLYQTVKEINEAEYTANLLLLQEYNELLQADLTSKQAVVLEHIHKHKRLTVKELADLMLITSSAVSQLIGKLDKEGY
LKREINPNNRREILVELDQAGWDYFERQQKVEKSIAERFYSKLDPEELSALRKITFKLQKIIEAELR
>lcl|NZ_CP026520.1_prot_WP_042227790.1_2770 [locus_tag=PC41400_RS14095] [protein=PAS domain S-box protein] [protein_id=WP_042227790.1] [location=complement(3127641..3130373)] [gbkey=CDS]
MKPFEDKWLFEIAYENAPFAMGVYSLAEDRWLRVNPSFCRMLGYTEDELVNLSDRRITYPEDIALSGADPSLFRFEKRYV
HKNGGIVWAAVGLSPLPPDSAEPAAFLVQAVDLTASKVREEQLLDIERQYRLITEASLDWISRHAPDEMATMTYSSPVCK
TFFGYEPEEMIGQAAIPLVHPDDLDRVLMFLERNRHTEGDKVTFRFRCKNGVYVWLESTTRYTYDKDGNVSEIVSISRNI
TERKKADQQLQESEQRYKSLFDHNPAAVYSMNLDGDYKTANKNLQKITGYTLEELIGMYWGPIVAPIDLPKTLHHFELAK
QGFPQSYDLTIIHKDGHPVEINSTNIPIIVDNEVVGVYGITIDITERQRYLEQIEKLSSEYTLILNSVTEGIFGVDFSGN
AMFINPAGADMLGFVPEELIGKPYLGLLQNTFGSPAQQSEDEQVIYRAIREGRSYHNKEAVFWRKDGSSFLADYHVSPLI
DNEELRGAVVVFKDITGEKEIIRAKESAEKADQAKSEFLAIMSHEIRTPMNGIIGMADLLAETELSEEQRSYADIILQSS
YALLRILNEILDFSKIEAGKMVVTHEPFDVRSVLNHVAELFHPKAAERGIRLTYELDGSIPSTLIGDEGILRQIIVNLVG
NAIKFTEKDGHVSISAKLDKIPGRSDAVLQFFVRDTGIGIPADKQSQLFQSFTQLHPTINRKYGGTGLGLAICKKLVELL
GGAIGVDSQVNVGSTFHFVLPFELPGESPAPDSAPEKTQRSKTASSLVSGRESGALPSPKFGPMKLLVTDDNPVNRLLLI
TLLKKLGYEADYAENGAEALQAVRSDTYDMVFMDLQMPEMDGFETTAAIRRMKGEASRPAIVAVTAFAQKEDMQMCLASG
MNDFISKPVFAPEVERVLKIWSPHCKQDRI
>lcl|NZ_CP026520.1_prot_WP_042227788.1_2771 [locus_tag=PC41400_RS14100] [protein=YheC/YheD family protein] [protein_id=WP_042227788.1] [location=3130695..3131432] [gbkey=CDS]
MNLGGYATKQNVSDSLLKQGEISRNVPETRLYSKGTLKTMLDTYKMVYVKPNSGTGGNGVIRVERKENGYSFQLHTFVKK
FPDFDSMAAALRKRIKKTPYVIQQGIHLLRHKGRLFDLRIMIQKNPSGVWETTGIIGRIGHPRKIVTNLCRGGSSKPVEV
LLKEHGADIKRYQASLRSLGLRAARHLNKTFPRIKEVGLDIGLDPNLHPWILEANPRPKIRGFKTLKDQSIYRKMLRYQK
YHGHA
>lcl|NZ_CP026520.1_prot_WP_042227786.1_2772 [locus_tag=PC41400_RS14105] [protein=O-methyltransferase] [protein_id=WP_042227786.1] [location=complement(3131549..3132274)] [gbkey=CDS]
MTLSNESPEVRNTPAQAEADLSGKWSHVDACLNEWLVPADPVMTEVLANNASASLPPIDVAPNQGKLLHLLAKIQGARSI
LEIGTLGGYSTIWLARALPADGKIVTLEYDPHHAETARQNLSRAGVSHLAELRIGEALVSLEQLYAEGAGPFDLIFIDAD
KPNNPDYFKWALKLSRPGSLIIGDNVIRDGEVANPHSDDPRVQGVRTFLELIAAEPRVTATALQTVGSKGYDGFVLALVT
A
>lcl|NZ_CP026520.1_prot_WP_042227785.1_2773 [locus_tag=PC41400_RS14110] [protein=LysR family transcriptional regulator] [protein_id=WP_042227785.1] [location=3132393..3133298] [gbkey=CDS]
MNLHALRIFYETAKAGSITGAAEVLRISQPAVTSQIKRFEKEIGMRLFAARGRGISLTDAGQKLLEYARKLFALEEEMEA
FVDGYTSGRNGAIRLVSTYLPANFLLPAWVAKFKRDHEEVDFVITTTNSRGAFDSLLNYEAEIAVYGGAGERQPGIHWEE
LFEDELWFVVPVHHKYANEHITLAEIVKEPFIMREEGSSTREWLFSLCRVHNVSPPQIALQFNGLNETISAVMAGYGTNF
ISSLVVRDHVERGALARVYVKDILLKNIIAVCTRTDEVLSPPAERFIQMIKASVGAGQALP
>lcl|NZ_CP026520.1_prot_WP_042227783.1_2774 [locus_tag=PC41400_RS14115] [protein=SMI1/KNR4 family protein] [protein_id=WP_042227783.1] [location=complement(3133388..3133972)] [gbkey=CDS]
MIKHHIDRLEKLLPLKVGEDCQEEYETYLKLEGAGEEQFAVLEMKYGVILPDDFKEFYRYKNGSGYHFHIFYPSYGEGAV
SPFYLLSLEEIDKLKAYFCSEDILLSEHFEEDEIARLDKRIRPYYGNRKWLPIAQLAGGSLYLMLDFDPSEHGKAGQVIA
FVHDPDFIYYAADSFAELLDISNKNLEDWEEIDY
>lcl|NZ_CP026520.1_prot_WP_042227781.1_2775 [locus_tag=PC41400_RS14120] [protein=GHKL domain-containing protein] [protein_id=WP_042227781.1] [location=complement(3134300..3136045)] [gbkey=CDS]
MSIKRLSVLFLAVIMAILLVNDSVFYFSTKKSLTANMETELFSLSQQIQLSIDHVEYSSQTRDKLLSDSLRQVALGVKAS
LPSRYRDIDNSMLAELRVKYNVSNLTLFAQIGDEIVSVRSSDPKEINLNAKYWGNYRIALEQLFRLETVTVSEGTTLQSF
WAGKISGSASNPGSKRNKFGFYYDGTTDYIINPTMNSSLIEQFDQNDGSEFIVAETLKNNKSLLEITGFNPRTFSKPDDD
PSFYTVTNGSKNIRYSERKILFGQYNSRSPEDVTLIEQAVKLDKPIFFPTKINDKNVLKIFYPVKSERVQFPYVIGLVND
MEYIHNSLNKQLLNFFLVIVSVSIASLIMVYFIYKGVNRTKDAVAQTIQENYTDEMNSLMVAIKGQNHDFTNHLTTISCM
VQLKKFEETKKYIHELVEETRLVNSIVHVGHPALASLISSKITQSQLRKIHFTYELDLADDLTSISGLKSVDLVKILGNL
IDNAFDEVSGMNVEQRNVHLLGYTEENRLIFKLSNHISKSLSEKDLQAMFTSGFSTKKDGVHKGLGLAIVNQLVQRYKGT
VSAEISGANTIQFTVILPIYS
>lcl|NZ_CP026520.1_prot_WP_042227780.1_2776 [locus_tag=PC41400_RS14125] [protein=extracellular solute-binding protein] [protein_id=WP_042227780.1] [location=3136364..3137989] [gbkey=CDS]
MSRKNRHKGLLPAAVIAAMLTVTACSTAAPEEAKTPETGGVMEVSTVRASDPYLKFDLGETFDNNAVYAAYEKETGVKIT
NKWVADGSQFAEKLKMSIASNEIPDFFPVDAAQLQQLIEADMILDLTPVYDKVATENTKKMMTMDGGLQMKSATFDGKLM
AIPSTSNPYYAQFLWVRTDWLKKLNLPEPKTMEDVAKIAEAFKTKDPGGTGKAYGLAFNKNLFDGALGVTGFLNGYHAYT
NQWLDDGTGKLQFGDIQPQMKDALRGLQDLYKKGLIDPEFAVKDATKASEMLYNDSIGMVYGAEWMPAQLASGAVKDGKV
VQEWGVFPLPSSDGKPAKSQIGLGVDKYYVVSKKAKHPEAVIKLLNHWIVVNNSQSEESKVYEFGKDKVEKSSNYWLLNP
IFVNNQTNNNGEVLPDAIANKDESKLQTKDQKTRYGRAMKYVNGDTSLWWEYLISGPNGAVSLYPEMKKNNQFEQNKFYG
APTPTMVEKGAILNKKRDEVFVKIIMNQVPVDEFDKFVADWKSMGGNEITKEVNNWYSKNK
>lcl|NZ_CP026520.1_prot_WP_042227779.1_2777 [locus_tag=PC41400_RS14130] [protein=ABC transporter permease subunit] [protein_id=WP_042227779.1] [location=3138100..3139005] [gbkey=CDS]
MHTRFRTWRREWPLHLMVVPGLIIVIIFSYIPMAGIMMAFQKFIPSKGLFGSPFVGLKNFRFLMEYPDIGQIFFNTLFIA
TMKIAAGLIVPITIAILLNELSRQWIKRTFQTLVYLPHFMSWVLLSGILIDVLSPSSGIVNQLLGVFGVQPVFFLGDNAW
FPYVMVVSDVWKEFGFGTIVYLAALTGINPSLYEAAEMDGAGRWKQTLYITLPGMMPIIVLMLTLNIGNVLNAGFDQIFN
LYSPQVYESADIIDTFVFRMGVQQSQFGFATAVGLLKSIVSFVLISVSYLLAYRFANYRIF
>lcl|NZ_CP026520.1_prot_WP_084706546.1_2778 [locus_tag=PC41400_RS14135] [protein=carbohydrate ABC transporter permease] [protein_id=WP_084706546.1] [location=3139018..3139902] [gbkey=CDS]
MRHQLTMGRRLFLIGNVVFLALISFACLMPIIHILAISLSSGAAASAGKVLLWPVDFTTAAYDNVFGKPEYLRAFWISVQ
RVVLGTAISMFLTILTAYPLSKDPRVFRLRTFYVWVFVFTILFSGGLIPWYMTIKTLGLIDTVWALVLPGALPVFNVILL
LNFYRSLPKELEESARIDGAGHFATLWKIYAPLSLPALATTGLFTVVGHWNSWFDGMILMNHPDKYPLQTFLQTIIINMD
FRFIKAQDAELLVKLSDRTSKAAQIFVAAFPVLLVYPFLQRFFIKGIVMGSVKE
>lcl|NZ_CP026520.1_prot_WP_042227776.1_2779 [locus_tag=PC41400_RS14140] [protein=histidine kinase] [protein_id=WP_042227776.1] [location=3139939..3141669] [gbkey=CDS]
MKLNLSLQDSSIFKKIMAVFMAALLPIMAITWLINEKGSESIRQERSQSILNTTGYYLDSLDKEADRINRFLPNYVTDSD
LQEIAAMGNELSDYDRSQKILAIQKRLNLTKDSSAFIQEVKAYVPLIQRTLLSVKYDTFVDLAEYEALQLRNKRADEPFV
FWDDRLFLSMQVPSNSTREPLYIVGVELSTAKIRQSLAQISGSFGGETLLINTDRGWNFQSGNDPQLLTAAKAFVGEKSA
EGNKQGYETITVGSDRYMAVYNYSALWNSYSVSCIPEKTLLGPIQTYRVWFRWACVLAVAVVLFFSFFLFRLIYRPLLKL
VQSFRRVQQNRLELIAVDRGNDEFGYLYHAFNQTIHSLRELIEQNYEQQIRSQRSELKRLQSQINPHFLYNCFFVLCRLI
KSEHKEKAYQFCLYIGDYFHFITRDDEDEIPLSLEINHSRTYVDMQKVCYGDRIEAVFEVEELDTRVPRLILQPIIENAY
KHALGSTPGAGFLRVWSRMDAKGITIYVEDNGTTLTDEELGKLRRRLADGNHELEETTGLVNVHRRIQLRYGQGYGITVD
RSELGGLKAGIHIGNS
>lcl|NZ_CP026520.1_prot_WP_042227774.1_2780 [locus_tag=PC41400_RS14145] [protein=response regulator] [protein_id=WP_042227774.1] [location=3141683..3143359] [gbkey=CDS]
MYRLLIVDDIPIIVEGLKDLFEETPELSLEIHTAYSGEEALEVLRNKRIDLVISDIKMPGLEGIGLLREIKTNWPSCKVI
FLTGYNDFQYVQNAVKYGGFDYLLKIESDEKIIAAVTRAIGKLDEEKAREEMIERAEESMKRALPSLQKEYVSSLLQGKK
PAGEQLDRHFREIGIPLQGEEPVLLLLGRVDVWREMLTPPDKALFVYAVQNIAEEILSPGARVYTYVYDQVKIAWFIQPA
ESSCPEKNVSGDNWDRTRRFVGNALETIQRACKQLLKLPVSFVVGKNPVPWDDISDKFHELKYNFLFGAGLGNEIILTEP
APDNHGETGHAQKRSEFFTAARVQLLGKCLENGHRCEFFNVYKEISDLWESDEWPFERKREVYHSLSAVFLAYVNKNDEL
RAYLNSSHDLSTLFHSEADMTAAALKAYFLQTAELIFDWNGEQAAHFPDELVQRVHTYIADNLAKDISLNAIADHVGLNP
SYLSRLYKQMTGIGLSDYINDYRNLKAKELLLGSPMRVGEIASMLGYNSALAFIRFFKKQNESTPQEYRSRRASGDSG
>lcl|NZ_CP026520.1_prot_WP_042227772.1_2781 [locus_tag=PC41400_RS14150] [protein=Gfo/Idh/MocA family oxidoreductase] [protein_id=WP_042227772.1] [location=3143473..3144750] [gbkey=CDS]
MAALTAVLLGAGSRGRYIYGPYAEKYPHELNIISVAEPDEGRRKLISRIHGIKPDYVYSSWEQAFERGKIADIMIISTLD
RMHFVPAMKALELGYHVLLEKPMSPKLEECVQLVGAAKAHDRLLVVSHVLRYSLFWAGIKQHLDAGELGDIASIQLTENV
GYAHMTHSYVRGNWRRAEETSPMILAKSCHDLDIISWLMGGECTSVSSFGSLFHFKKENAPAGSTDRCIEGCVVEKECPF
SALKLYIQPPDHPWARYMTHDLSHEGIVKALKEGPFGRCVYRCDNDVVDHQVVNMEFAGGATAAFTMSGFTQGGGRRVQI
MGTRGEILGDMDTGTYTLHRFTTGEAVEVRCNIAGDGHGGGDERMVASFLKDVREYGRAPAASRTSASASLQSHLMAFGA
EESRLNGGKSVSLADFKRASAAGIV
>lcl|NZ_CP026520.1_prot_WP_042227770.1_2782 [locus_tag=PC41400_RS14155] [protein=Ger(x)C family spore germination protein] [protein_id=WP_042227770.1] [location=complement(3144864..3145988)] [gbkey=CDS]
MNRFSKTAASLLTLLLLGGCWDQQLLKSSRIIHIGGFDRGEGDELKVTVAFPDVSSTEGGRSETNELHTVTSNTTQQARG
FLDHEISGNYSPSKLLVLLLGEEWARAQNIMPYMDVYYRDPKSPLNARLAVVNGTAEDLIGLKKVGTKLLGRFLDALLEN
MEDGTVIPRINFQSIGPVKPGQDFALPYVSIHNKKPVVQGVALFHSNVMTASLNSEESLIYLLLKGQRGKTARLTLPVNI
EEEEEHETKADRYITVDLHKYSRKIKVELTESGGVRVKIQLKINAAIIEYPLDRLNKKETLQKLNAELSRLLTARAKAVT
RKMQEANHDGYGVALRLMAFHPEAWKKMNWEEDYKEVQFEPDIRLKIERKGITF
>lcl|NZ_CP026520.1_prot_WP_042227768.1_2783 [locus_tag=PC41400_RS14160] [protein=GerAB/ArcD/ProY family transporter] [protein_id=WP_042227768.1] [location=complement(3145985..3147088)] [gbkey=CDS]
MTHNSNKISPKQMLFLVLQTQIGVQVLFMPSRVQAFAKGDAWMSVLLAGLFSQLLILMMWALNKRFPSATLYDFMPRLTG
KAAAAFLQVLYIVYFTVQSSLILMFFYEVIRDWVFPETPEWLITGLMVLACIYLTQENLQIIARFFLLVSIVILVMIFIS
VYAYTHSNLLYIFPIGQAGFLPVVRGAHEAMNTLTGYEWILFCYPFVEGTPRRKLAAASVANAFSTLLYTFLVFTSIVVF
SPSELGLLPHPVLYMVKALTFTIIERPDLYFLSAWIVVVATSIMASLYMASTGISHLLGMSNHRKPVLFTAIAFLAIALL
IHDQDVMKTLGKIVSLSSYFFLGFVPLLLLLISLALHIREGGRRLVK
>lcl|NZ_CP026520.1_prot_WP_042227766.1_2784 [locus_tag=PC41400_RS14165] [protein=spore germination protein] [protein_id=WP_042227766.1] [location=complement(3147069..3148613)] [gbkey=CDS]
MKLFRHLFHMNKKSAPTPASVQETADTVTIKSSLAENERYLDQALFHTDDLVKLTLTYQNQNLLLCYLDPMCDEVKIRSF
ILKPLAESEEENLADVFTAPRVYAQTSLDEAADTLLQGQAVLFKEGSTECYTITVNAEVKRTTEEPENEKTVRGSHEGFI
ENMTANLHLIRKQIKNRHLVIRRYTLGKESRTEVALAYLNHVAKPELILEIDKRLRTIAGANTLSTDLMEEYIEDRTLSP
FPQILKTERPDRAAANLLEGRAVLISAGTPSVLIMPVTFFSFYQSPDDYYSRTLAGSFIRFIRLLSFVIAITLPAFYIAV
ITFHYEVIPAELLFQIKGSVEMVPYPPLLEAMFMELTLELLREAGIRLPGPIGQTIGIVGGLVIGDAVVRVGLVSYPMII
VVALTAISSFIVPSHEMSASVRLLRFAVMIPAALLGFIGIVFSLIFILIHLCKLESFGTPYLAPAAPFRLKDWKDTIFRF
PLWKKDNRVNMAKPSVKKDNADSKELQENDTQFE
>lcl|NZ_CP026520.1_prot_WP_042227764.1_2785 [locus_tag=PC41400_RS14170] [protein=NAD(P)H-dependent oxidoreductase] [protein_id=WP_042227764.1] [location=complement(3148829..3149368)] [gbkey=CDS]
MKIMVITAHPDLTASRVNRAFARKASTVPGVYVRDLYREYPEWTIDVEKEQRLLTEHDRIVFQFPLYWYSSPALLKKWFD
DVLTYGWAFGTDGVHLTGKEFIVATTTGGTENQYRSGGSNWHTLSEFMKPIQSTIEKCNGVFLPVFAAYHLSDAPEAELE
AEADRYASHIVSLQADLAD
>lcl|NZ_CP026520.1_prot_WP_042227763.1_2786 [locus_tag=PC41400_RS14175] [protein=helix-turn-helix domain-containing protein] [protein_id=WP_042227763.1] [location=3149625..3150002] [gbkey=CDS]
MPLNDRQPGTTEGIMTTLDVIGGKWKPLILFILLHQGTKRFGELRKLLPSVTQGMLTNHLRELERDGLIVRKIYQEIPPK
VEYSMSDHGRTLSTVLADMCGWGFMHREVIESGGQAERSRTGESD
>lcl|NZ_CP026520.1_prot_WP_042227762.1_2787 [locus_tag=PC41400_RS14180] [protein=DUF418 domain-containing protein] [protein_id=WP_042227762.1] [location=complement(3150786..3151985)] [gbkey=CDS]
MSDRFKMPDENGNEKEKGTGSGHRRPENAFPNRRGNPQGAGRLLALDAARGLAVIGMYLQHFASNERITTIVSGNSTLLF
VLCGGISYSIMAQRMKDRQKDATAFRAKMLARAVFIDIMGYLLIMLNTPFGVILPAYAAMFVLALILAGHSNRTLVTTTV
ALMFLAPPLMLLGMSALSGAFLLRDVAGGPMSTLALAPAFVAGMAIGSLELTKKRTAFLLAIVGLMLLIIGKTLAANVLP
ELNRSFESWLVTVQGIAAMPDPYAIWPLNMEQPMWHMLLWTVPHSASTFQTLTGLGFALLVVGIVFLVPKKTDAILVPFA
GVGRVALTMYAAQFVVVWLLTLSSIDYSLAGVPLGDLLVALITLVAGWLISRLPQGPLESSMRHFDRLFSVPRSITAPK
>lcl|NZ_CP026520.1_prot_WP_042227761.1_2788 [locus_tag=PC41400_RS14185] [protein=response regulator transcription factor] [protein_id=WP_042227761.1] [location=3152329..3153018] [gbkey=CDS]
MNQAKILIIEDDAAIADLLSYGLSMEGFETKTTASGAEGMMELQQFQPDLLLLDWMLPDQSGLDICKKVTASHNLPIMMI
TAKSDITDKILGLEFGADDYITKPFDLREVVARIRTILRRVDQANKRHDDQREVIRLEHIEIIVEERLVQKDGHTVDLTP
KEFDLLMTLIGHRGKTYTRAELLDIVWGYDFHGDTRTVDTHIQRLRKKLDAGDLIATVFGIGYKFENRR
>lcl|NZ_CP026520.1_prot_WP_042227760.1_2789 [locus_tag=PC41400_RS14190] [protein=HAMP domain-containing sensor histidine kinase] [protein_id=WP_042227760.1] [location=3153024..3154487] [gbkey=CDS]
MIRNISVKFIIGFFFIFSLSFLVLNQTVKESIQTSNKKIVTSELIGLKNNSNIYVRQAFLINHFSNNRLYFGQMAEEMAN
DLNHATGSSVGVYTVDGELLYSSDKSIPPLTNGDLKEAIEGKTAYSITYDRNKASVLFSYPVFIDGTKVGILRFSRDFSL
LYEQSGRIREIIFYIAMAIFAAAFLFSYILSRHITIPLVRLTRASNEVKSGNLEVRIRFRRKDEIGRLADNFNEMIDQIS
RQMTIIRKDRDHLQALHEQEKRFFDNITHELKTPLTSILGYAEIIRKKGETDRYFFEKGMNHIIEESRRLHDMVLKLLEA
TRQNIQIMEPERVDAGSILKDVCESMGIRAQRYKKSIRYEVQNELLVYGQGNRLRQLFINLLDNSIKYSTDQAEIFVTAK
RADTKIQFVFSNPSDPIEADQIAYLFQPFNLGNRQISEEGSVGLGLSISKSIVDEHGGNIRMTNENNQTVVSVELEYMNV
RKDGKPI
>lcl|NZ_CP026520.1_prot_WP_042227759.1_2790 [locus_tag=PC41400_RS14195] [protein=hypothetical protein] [protein_id=WP_042227759.1] [location=3154484..3155623] [gbkey=CDS]
MKNALFAIAVLLFLILVLSGCALFEPETETIVIPSQEEKQTENEGIQPFKVKTIYRLPDEFTNAGQLLGWSGKDSVIASN
KQVNMSEQTVLKRMAPPYEKSEIISGIQMDSPVIELSPDGSLIREISITRFGTKLKLISLKDGKETEVANFSNTKDLYLQ
NAAWSNNNQYLCYLVIDPIESGKAFVHLLNVGNQSSTTYTIKNYGKGNNLTKINVSDDGHSVLLTAFQTGKTKKNIIMLG
TVNGNSIDIQYEHQSDENQVAWLNNDQFVFLGTDGTLFEYDQRTKELSVLLEKVEGFKLSKDRKCIAYSTYDQDTIYAGK
IQGNNVLSNEPVYHGLKPSQMYWSPGEKGLLVYGQKFYSPSSNTQTMTTDGQVLLIEFQ
>lcl|NZ_CP026520.1_prot_WP_042227757.1_2791 [locus_tag=PC41400_RS14200] [protein=acyltransferase] [protein_id=WP_042227757.1] [location=3155834..3156958] [gbkey=CDS]
MGKQKQLLKRYIPGLDGLRAISVLAVIAYHFNLKWAKGGLLGVEVFFVLSGYLITDQLLWELKTQRKISFLHFWIRRIRR
LLPAMVSMLLVVALGLILIDPSRMQTLRGDFLSSVFYVNNWYLIFHQVSYFESFGSPSPIGHLWSLSIEEQFYVVWPLLL
FFLVRLLGRRGRLAVFILSGAAVSAIAMALLYEPGTDPSRVYYGTDTRAFAILIGAALAVVWPSWRLTDRISSGARLLLD
LLGAVAMLILFIMMNRTNEYDSSLYPVGFLFLSVVTAVIIAVLVHPASQLGSILAAKPLRWIGKRSYSLYIWHYPVIILS
SPAANVGEPGFVDILLQLTLIFMLSALSYRFVEEPIRRGRSWWTRSKELNKSTL
>lcl|NZ_CP026520.1_prot_WP_053228667.1_2792 [locus_tag=PC41400_RS14205] [protein=hypothetical protein] [protein_id=WP_053228667.1] [location=3157001..3157672] [gbkey=CDS]
MKKHKVGITWVFIIIILAAGYFSQRHQRSKTTEQGELGGQQITAPQQIKASSESAPQKTENKKPPQSLDGEGITVIGDSV
IVGIEPYLKEKLPKITVDGKVGRQMSQAQKLIDDLKDEGKLGNRIIIELGTNGPFNNKSLRSLLQSLSDKQVFVVTTRVS
KGWQDTVNTTIKDVAGEFHHVKIVDWYAASEGKKDIFYEDGVHLKPDGSRYYASLLIEALQKQ
>lcl|NZ_CP026520.1_prot_WP_042227755.1_2793 [locus_tag=PC41400_RS14210] [protein=LCP family protein] [protein_id=WP_042227755.1] [location=3157711..3158739] [gbkey=CDS]
MSLKTKRIAAVLTLILIVLSGAVVCYFYSLYRGLEGLNKTDSQKSLFAGVNAAEAEYTEPPKWLGTEPVNILLMGVDARG
VQKGEIPRSDTMMVVSLDPERKRIHLFSILRDTYTDIPDHGKNRINTAITYGPNTAMKAVSDLLGIPVQRYVYTDFQGFI
KLVDEIGGVDFYVEKDMNYESAADQHAYDIHLTEGQQHLDGKAALQYVRFRHDALSDYSRTKRQRDFTDAVARKMKSTTS
IMKLPSILEKVSPYMDTNLTVNDMWKLASIGYQSTLDGSEQIPPMKLIKETYRGQAAVLDVSSRKELKDFVQNIMEAEKS
IPAETPDSLKSGESGVTAGMNE
>lcl|NZ_CP026520.1_prot_WP_255246830.1_2794 [locus_tag=PC41400_RS30585] [protein=hypothetical protein] [protein_id=WP_255246830.1] [location=3158757..3158882] [gbkey=CDS]
MKKKLNKHALGLMISFSLMIGTVAPGDVKAEPISRIKQIEG
>lcl|NZ_CP026520.1_prot_WP_129112480.1_2795 [locus_tag=PC41400_RS14215] [protein=hypothetical protein] [protein_id=WP_129112480.1] [location=complement(3158951..3159397)] [gbkey=CDS]
MKKTLFSTTAAFVLLGSASSISYASPMNLTTIKNPPVISGVTPLDLIKDEHTNFNGESSSRFYYTTGKDLRIFLKNEGST
SFSYTVKLPNGDVLDSGNLAPGQQVTENHYWYTSPEPSPLQTGNYTVYIRNSDGSYGSYWIAVRSLES
>lcl|NZ_CP026520.1_prot_WP_042227754.1_2796 [locus_tag=PC41400_RS14220] [protein=hypothetical protein] [protein_id=WP_042227754.1] [location=3159717..3160091] [gbkey=CDS]
MAFKTEYEFELPRGFEDSSGNTHKKGVMRLATAADEILPMRDSRVQANPQYLFVIILTRVITRLGDLPMIDTKVIEQLFI
ADLAFLRDLYLKINKDEDPIIPATCPICEHEFIVPIPVGLKWED
>lcl|NZ_CP026520.1_prot_WP_042227752.1_2797 [locus_tag=PC41400_RS14225] [protein=metallophosphatase] [protein_id=WP_042227752.1] [location=3160110..3160805] [gbkey=CDS]
MIYITGDIHGTVSVGSRLNKKNFPEQKYLTKKDCVIIAGDFGLIWDGSREDQYWLKWLHKEKPFTTLFIDGNHENFDLLE
QYPVEIWNGGKVHKINSSVIHLMRGQVFTIENKKFFTFGGAASHDKEYRKEGKSWWRREMPSKDEYEEGLENLNRNNWIV
DYVITHTCSTSDLDYISSVFNIHMEKDEMHPYFYEVEQKLKYKRWYFGHFHQNVELPKNKRLIYSDMIKIQ
>lcl|NZ_CP026520.1_prot_WP_129112481.1_2798 [locus_tag=PC41400_RS14235] [protein=hypothetical protein] [protein_id=WP_129112481.1] [location=complement(3161141..3161527)] [gbkey=CDS]
MVAPQDVTGFSYEFSDFGRASDPLLTKGTTTFNTTAAVYIILVQWSTDYYNGATGKVDMLYELIDVKTNKPVQSERVYEQ
LTSQNKTIGWSNVPAGSYKVRLTNKGAGYGHGNGFVRVLDEPRKDQFL
>lcl|NZ_CP026520.1_prot_WP_042227749.1_2799 [locus_tag=PC41400_RS14240] [protein=OsmC family protein] [protein_id=WP_042227749.1] [location=complement(3162064..3162507)] [gbkey=CDS]
MATTTFKASAHLQEGVVVKVSSRNFEFTIDEPKSLGGTDTGMNPVEVLLGSLGACQSIVARVYAKKFNINLEDFRVEVEG
DLDLDGFFKKADVRPGYSDIRYTFYIKTDAPRERVEEFVKFLEETCPVGDTIANLVNLKLNGIVIEN
>lcl|NZ_CP026520.1_prot_WP_042227747.1_2800 [locus_tag=PC41400_RS14245] [protein=SUKH-4 family immunity protein] [protein_id=WP_042227747.1] [location=3163088..3163669] [gbkey=CDS]
MKNAVFYDPNLVRSVNLPEKIEAFLLKEGLPLIELAGAVLGVRFQSFTDMGLLYDGERKLLPIGYEWEPESSILGLAQGS
GHLYSFHPASGQSGFVNSDICLFLLFLSCVRSFIASRSESDQATIWTAEEMRERLAAFQRGEIVPKRPNRPVFNRKAELT
RMRAMFEEKDAASLADENHWWSCVLEQLEDGLL
>lcl|NZ_CP026520.1_prot_WP_042227745.1_2801 [locus_tag=PC41400_RS14250] [protein=hypothetical protein] [protein_id=WP_042227745.1] [location=3163861..3164301] [gbkey=CDS]
MKKWIVLFLSVAVCIYVLSFLNLFNGKPLPEAPRAYIKSASIDIPVETASYYWKFGKEKNMSPWYAVKDKEAIIVPKNSV
IKISFKREPDGPPIIRQVITDNDNRDIETKNSNEITVPSSEGIYMYSISADWSEGSAQYAIKVHVK
>lcl|NZ_CP026520.1_prot_WP_042227743.1_2802 [locus_tag=PC41400_RS14255] [protein=hypothetical protein] [protein_id=WP_042227743.1] [location=complement(3164544..3165020)] [gbkey=CDS]
MKKSIVSFLSAALLLSSVSSVALAEQNVMDPQSLVTPQAVGADSGWQTQNNIRARVYTDDTFYGTGANTISVTVEKSTPG
SQFYLVYFKNDTHMVEVARGTLTTKATHNISKSSLNSILPQNSQGNFKVLLKFMEPGPEAPKWLGDWFTKNTFTAVRI
>lcl|NZ_CP026520.1_prot_WP_129112482.1_2803 [locus_tag=PC41400_RS14260] [protein=hypothetical protein] [protein_id=WP_129112482.1] [location=3165279..3165938] [gbkey=CDS]
MRQFQLPRIGPSFCWAGFDGPTGANINIGAVRYSQANPEIKIITTSWVDIPYTSGGGTLGRLGLRSNDTQTKLFKESGND
FPATEWSKFQSTRLPIPMSGTSEFEQTRGVNYSVWNGNDGGTYYMTWETNEGPWSEFNLVTAGLGMFQNGKPMSVLYTNR
APSSPFVCQGYWIQVGSPFTPDMGLGRVKFNWNGRDKMTRNGFEGSGNPPTELTLKGNR
>lcl|NZ_CP026520.1_prot_WP_042227739.1_2804 [locus_tag=PC41400_RS14265] [protein=hypothetical protein] [protein_id=WP_042227739.1] [location=3166497..3166874] [gbkey=CDS]
MFNKKLAIVILSAALTFTLSSVAFAIDEGTYAAVQGNNTASTQYITGQDKTGRFIIKASGGTADARLYEKCTYNWSQIES
LSATQESTTGYAKKSKDVYLKNGCTYKADLYGSGSVSAGVELFNY
>lcl|NZ_CP026520.1_prot_WP_129112483.1_2805 [locus_tag=PC41400_RS14270] [protein=hypothetical protein] [protein_id=WP_129112483.1] [location=complement(3167176..3167733)] [gbkey=CDS]
MALIPVTIFANESTSDNSTVYAVPPGGGRVYPGTNVSVIPGDVLVTNDTRSMGLHGHAAIVGDWDGVTVSIEGSGYSPGL
TSLDDWFKKRPNTKVMRWTGASSQAINAGFWAKSYVSTHPNATFSLINTLGDQDKVYCSKIPWMAFYYGSNIAIDGHSYI
ADMVYSPYAWLSASKMSAVAAIGAM
>lcl|NZ_CP026520.1_prot_WP_042227738.1_2806 [locus_tag=PC41400_RS14275] [protein=DUF3139 domain-containing protein] [protein_id=WP_042227738.1] [location=complement(3167866..3168201)] [gbkey=CDS]
MKTIRKIWLIVLIVFMLAVGFFIGDFYGSEIKINTAISQSLEKRTVEHLSIKGFESKDYTFKTVKAQKFVGINDYAVAVQ
FNDEAENTYYYSEDTDGKIYQSSRNGQKHIE
>lcl|NZ_CP026520.1_prot_WP_084706543.1_2807 [locus_tag=PC41400_RS14280] [protein=site-specific integrase] [protein_id=WP_084706543.1] [location=complement(3169210..3170319)] [gbkey=CDS]
MAIYEYSKRGKIHYFYSFEVKDFHGKRKTIKQKGFTGKREARAAEAEARVEWEKGLFVDPSTITYAEYLKSWITNKHNIS
PQTRETNEGQIKNHINPVIGNVLLQKLTTQHIEILITSLKNKSLSSGTVKKIFNLVNTSLKSAVKKELIVKNPIDLMDVG
SKPAVKKGRVDFWTVDEVKDFLDGLEHRHRIIFVLAIYTGMRRGEILGLRWRDIDLENGQIRISQILAFKGEIKDGAKTV
SGNRSISIPPIVVSELRKHRSTIIKEKWDAQEKYKDHDLVVCRKNGLPVSWSNFHKFWLRILSKSNVRPIRFHDLRHTCA
SLLLSGGIHPKIVQELLGHSSIKITLDTYSHMMPNIQADAVKTLEKMLN
>lcl|NZ_CP026520.1_prot_WP_042227736.1_2808 [locus_tag=PC41400_RS14285] [protein=hypothetical protein] [protein_id=WP_042227736.1] [location=complement(3170525..3171160)] [gbkey=CDS]
MKKANEGKKFEKNYENSVNSTSHFFLRLKDAPKWLRGSSATFTPENPCDGIQYAAPFLWLVELKSTKGSSISFNPERPEE
KPRNKTTHVMIKAHQVKSLMKFATKDGLIPGFIFNFRERGTKTKYYPPETFFVHIYDFVKWTRNSKKSGINRDDCRTIGF
PIQTTLDKDYDICKFIDDSLPSFISRGYLKIEFLKSVVNRLNLLIGDSKKK
>lcl|NZ_CP026520.1_prot_WP_053228665.1_2809 [locus_tag=PC41400_RS30300] [protein=peptidoglycan recognition family protein] [protein_id=WP_053228665.1] [location=complement(3171162..3171950)] [gbkey=CDS]
MSFKMKYSITPLYLPTKTKRRTGITANKIKFMVAHDTGNPNSTARGNVKYYNNSANQIEASAHIFVDDKEIIECVPFLTD
KPEKAWHVLYNVDVDNKLFGCDANDAAGGIEYCYGTNINSNEAYKRFVWVLAYSFYKFNIQPTELIGHHILDPKRKTDPE
NGLSKSGRSYAQLKKDVIKEYYDCLNCEEADENMSLQLNDYEWKMLNSIWSQRYNKGEITDWKWIDRIRCRQLTAGELAF
LNSVVAAKQDCLSVEAESAGMQ
>lcl|NZ_CP026520.1_prot_WP_042227734.1_2810 [locus_tag=PC41400_RS14300] [protein=phage holin family protein] [protein_id=WP_042227734.1] [location=complement(3171964..3172365)] [gbkey=CDS]
MNESILKLIFSTGATITTYAFGGWSAVLGVLVALVVIDQVTGIVASIKDGTGLSSRVGFIGISKKVFIFVVIAMCHWFDG
IMNLNGILMDAAIYFYIANETISIIENLGRIDVLIPEQLKQIITVLRNKNQNK
>lcl|NZ_CP026520.1_prot_WP_084706542.1_2811 [locus_tag=PC41400_RS14305] [protein=XkdX family protein] [protein_id=WP_084706542.1] [location=complement(3172446..3172586)] [gbkey=CDS]
MDWFKTVKVYYDKGFYTNDNVKVFVVKNKITSEQYELITNEEFSVE
>lcl|NZ_CP026520.1_prot_WP_042227733.1_2812 [locus_tag=PC41400_RS14310] [protein=hypothetical protein] [protein_id=WP_042227733.1] [location=complement(3172608..3172892)] [gbkey=CDS]
MEKRYITLNADGEIGTIYASLKYAPKDAKPIEVDYILQDPEKSKTHVATLKFDPDKKCLYYEYTLAPPSQEQIIQDMQAA
QAVMMLALVDAGLI
>lcl|NZ_CP026520.1_prot_WP_042227732.1_2813 [locus_tag=PC41400_RS14315] [protein=hypothetical protein] [protein_id=WP_042227732.1] [location=complement(3172909..3174489)] [gbkey=CDS]
MSIRRLGDMRIKPGASGDVQSSDVRAGKTFSSEIDTDQVGTLPDRGAMTLTASGTGQVVIPDGIHSNSKIAQVNVPADKV
LTGTAIASVAGTMPNRGAPTFNVSPNNQTITPGYYTGGTVAGDVNIRPENIKKGVWIGGVQGNSLEYGAGDWISFGRIQK
YRLLQRWEKTFAGSNYVLEGRPSCVDNRYVCSMSYYDGTNRFTLLICWDVTGTEVWRHTVPAGGSNYALEGTSLISRDRA
HVFWSYGNSSGNTYSPKLRKIRLSDGATIWDNVEYIDQTAGALQETADGKIWLIGSFYTSTTYSAITRYSASGTLEAKQT
HSPGNGPQIVVQPDGQYYYLFDDNRRLRKFQATGTTSVDAGGNYFGKEQWSLSLSAIFRSGQGAFWAPALGQVCVSGGSY
AYANNGEAVNTGALAPQLAFVNQSGVLQGVSGVPCTSSTGVCTPINDGEFYFMGDSSVVTPYILRTSDKAVVARFDPGDP
NAFGANRSGKSFSWDFGNCAFAITRPIPNNGTTQYLSLHYLNTYFG
>lcl|NZ_CP026520.1_prot_WP_042227730.1_2814 [locus_tag=PC41400_RS14320] [protein=hypothetical protein] [protein_id=WP_042227730.1] [location=complement(3174486..3174815)] [gbkey=CDS]
MINTPKRLYVNLPGTTITTAYTVPALTTTIVKNIHIANNTSVVSTVTIHLVPFGQTAANSNKIIASYPVKPNDSISVDLS
SVLHSGDTIQVSQGTASAISVYISGVECV
>lcl|NZ_CP026520.1_prot_WP_042227729.1_2815 [locus_tag=PC41400_RS14325] [protein=hypothetical protein] [protein_id=WP_042227729.1] [location=complement(3174817..3175434)] [gbkey=CDS]
MAVNLSTFPVKIDEFQRHYELQAQDIPGIQRFQELKLKANRTPAEETELANLTVQYREKFISADDFNRFQDALVNMEIFI
KDNVEGYILTKQGEFTTYVDAAKNTVQATKDSALQTIEQKKENVITYLDGTTAGALRNDIGVMGDLETINKQSLVKVANE
LRSDNIKVDNKIGILSGLNSTDQSNIVKAINEIKSNLEIAYWMGV
>lcl|NZ_CP026520.1_prot_WP_042227728.1_2816 [locus_tag=PC41400_RS14330] [protein=hypothetical protein] [protein_id=WP_042227728.1] [location=complement(3175439..3176446)] [gbkey=CDS]
MATQKPYNINLKGTTVDAKDSIKVSWQSSGDMSVAFEVRIYRNQDNNLLYDSGRITSFAKNYSLPAFALPNGYELKVQVT
IWDSGNNFAVSDTEIFQTSSRPNVLVSPIGVVNSRSYEFSASYTQTENVPFRSWIAFLYDKDKTLISQSNVRTSTPLVYL
INYLKSGSSYFIEFQATSNKGSVGTSGLIPFSVSYLQPDISINLEAKNIKNSGIQLSWKTIQIIGNTTKQPIYLNGEKLD
LRDDVFTFTDGFHVDANFTLKLWLENIKSYTDLLVLKGQNGIMTLQYWDDDQRFHLFREVYNYRSHYSSQPLTGNGYFLC
IQQIGGDMNIFGEVN
>lcl|NZ_CP026520.1_prot_WP_042227727.1_2817 [locus_tag=PC41400_RS14335] [protein=hypothetical protein] [protein_id=WP_042227727.1] [location=complement(3176465..3177808)] [gbkey=CDS]
MTNEDFIKAVRQIVQLELKEMKLLNGHWHLGTVNKVKTKTLLSINIDGSSQTQDVPCNPDVIFNDGDLVYVVYVNGDSKS
KFVPYKRFADKNYETMVLANKVLGNLAIGSKEDFTKNAPFNFTVIEGIRGGVEYSELWILINAHYDFEAKRFKRKNINYN
SFGLQFQNKGTYPGEENYGDYTNQSIGIWRTSGRNQYSQNDPTRNLVGEEIGAFDGNGKWQEFGIMLGWNNTFMFDSYGG
TTVGGAGFEIDGNGTSPYKRVSLGKFNGGSLDSNKKYEEYGYAYNGDLWNSFHGMFGSDQGNRNSSFFGQICPIDFYDTG
QFNPYSNQASMDNLRLAWIVKPKNLGHQIENWKDIMSVDMDGNMKLKERTVSATLVANATVNGTDFNFNYPDSTWNKSNT
FIAGVIGVQSNGSLKQFGAINATFTDFGAYGYLGEGFQSAKILISKY
>lcl|NZ_CP026520.1_prot_WP_042227726.1_2818 [locus_tag=PC41400_RS14340] [protein=hypothetical protein] [protein_id=WP_042227726.1] [location=complement(3177821..3178999)] [gbkey=CDS]
MQYCSSEYLAALKSPIKQMYLKFEFYDSQMNYISEFTKQITAQDSGSISVDASRPVRRSFSFKLLNKKNEFNFGEDNLIW
IDKRVKVYTGLKLRTGEVEYIPQGVFVLSEPEDSHTSEGKLTSLSGMDKMFLCTDKRGKFRTETTIEVGGNIGAIIKLLA
QKYGETQFNFDPISETIPYTFTFQPSDNIYKAMSDLALLAKCELFYDVNGYLRLRKIDLNQFEQSPSVWRYTYGDLSERF
YAGNVRKFDENQLANRILVLGGSSQIAESRFELVVTEADPLWQGHPYSIEKIGEITYEHNSGNPDSLLSTNEECKWRAKY
ELMNRLGYAERVSLSLSPNYLHDVNDVIEIEDEENSVTGKYLIRSFSLPLNPQLMNCECMKYRRVITDWNFI
>lcl|NZ_CP026520.1_prot_WP_042227724.1_2819 [locus_tag=PC41400_RS14345] [protein=hypothetical protein] [protein_id=WP_042227724.1] [location=complement(3179011..3179865)] [gbkey=CDS]
MTIIGYDFYGTQFFRSPGNAIKDIFYGEIKGGIYDELEIRDRTDTPTDKGKTKWEIDHRAIFKFAGDLEGGNISNSGLEI
VSFAIKRRRVNEIDSVTLDHIPFVNNLSINYEDYTQPNDELIYTIVPIAQNDLGGKPVEVHVKSDFVGWWLVDKETDAIL
GFDSAIGEIGSVETALNQGRTVIETLSRFPSVYYFDKDYHTFSLSTAIIPSEFERSGQNYEKIVNTFIRDHRPFIVKSDT
GRIFVCDCSNPRFSTPLNTWRDYDYGVLTIDFLEIQDYNEFMNE
>lcl|NZ_CP026520.1_prot_WP_053228664.1_2820 [locus_tag=PC41400_RS14350] [protein=phage tail tape measure protein] [protein_id=WP_053228664.1] [location=complement(3179911..3186237)] [gbkey=CDS]
MSNDLRILISAGLNIGKSTNEINNAVKLLENKINNLKLNIQIDKNLSKGIQDLIVSVEKMKSISQDQNKVVSESQVVHKR
LDGSVQTLTQKVLKNGEVISKTRTIHDENKRAIQDEETATKKLTQGVEQLNKAELDSIKISKNKADKINGYTVKSTDGYK
TTTNRLESDGSTLINSQAVTNYKKQREDLIKEQERLNKKSEELDRAHYSALKDARKREEELEKTHYLALQQNQKRDQDYL
QSKANMQNKLNDIQRRFGADSNVSNQIENLNNQLSNVSSIGNYKNALSSIDTELKKVVTSAKTAGSNIMGFGQQFAVAIQ
RIPLWMGAMTAFYFPLRQFQEGLKSILEIDTALTNLKKVTDETTATYAQFVRESGEIANSVGNLTVNVIQSATEWARLGY
NIQQAKVLAKETLVYQNVGDIKSAEDASKSLISTIKGFGMEVDSQGKNVRKIVDIYNEVGNKFAISSAGIGEAMRRSAAS
LSEAGNTIEQAVALATGANSTIQDPARVGQALKTISMRLRGISEEGEDLSNLVPSLEQKFQSIGLTLQKDENTFKSTYEI
FQDLSKVWGKLSEFERSDILELVAGKLQGNIAASLISNFKDAEGALQAGLNSFGSAAKENETYLKSMQGHIAKLKNEATN
FWQGVMDSGALNQIIDLGTSLIGVLRSVTDTFGGLGSLAGVATAALIAFNVSVRSNTVGALSQSLANAALQFKSLSWFTN
LYTSSAVGARVATIALQATMTMGLSIAVAGLISGLTSLYQAHQKNKDIQDQVKAQNEQIAESWNTQQDKIQPLLSQYRAL
NALTNEGKTFSSPDQEQKYLQTVNDLSTLLPNLVARIDEKGQKHLLNADAIERELKYTKELAQLERQGKVDNAKSDFSST
IEEIEKAKKSLNKLKEDLVGEYLFVNKDNIKNIYGDVNAVQIKILNKENEIALKTDVLKNKMQTVVESFLKLNNIDIPAE
ASKNIATVIKSLDVSNFKDSEKMTQTVQNFSSFLKLLKENSGTEEAKNAFNALKDTLGLTDSQVSLLRSSFDSAKGSLDS
AANSAVNLNTVFSSLTGNYKAASDQIKPLNQLIDAMGKGKKMTADEVAQLSFQVEGFTEAIIDENGQIKNNVDAVINLRD
EKIQAYKDQEQGMIDNLNKLEQTTIQELKLYGIQVEGIKSVAEALGEISTKKSQVSADIRQMGRENDMAGVEEATKTFDV
LFDTEVKLGKIEAIKKRIEGLANTTRSGLEQVGMSEPAKDKKAKEKKAKTKSEPSPYDESEKYKINEYERALQRLDDELR
ESEAIINRYPDTSKDYRDELTKQIEIRKKMQALAHNEAERLRSEISDVKNKIAVGKLNQKQENDLLKQLEQKENNLSQLS
NKWWDYQVAVEATQKKITESMAKYNKEQFDISKNWIDQEKYYKRLSLDDELAAWERVQARYEQGSEERLNADKEVYRVRQ
ELLSKYEEDFNKVMEKTSQATDETIEKLRKKLETKETFNPSELTDSVDAIMASIDMIDRKYPNGFSPIDTSDDARVRLNA
YKANVTALASNMKSMMEMPYNNADQLKNMIDGQIMFAGNIKKLMNETNSLIKQTETAYRMQEASLENQIKLHQQRNDKII
EGLNKALEVKDTFNTTDFSTSINSILSALDALDGKYVKGNFVESTLGTREDLESYKKKVTEIAEAIKNYQNSADFSSSSI
TEQTKYINGLKTQIDSLNDSIRVKELQYKNEEEALVQLIKAKEKSYDDEINKQKETLKNLDEQIDKEDRLKKLKDINDEL
DKVKNDKRFSYINADGKEILTYDKGKVAELEKQRDDLLKQYEREDVKKALQDSIDQLEKTKNETIQKLQEQLDKTKVIHQ
NELESQKLYVKSLEDLYRIGLQDVQNKIGQLKSQYDTEIANEQARLDTLRQVHQSELDSMNMYAGYLSQLNQFLTQDTQT
KIDALKTIYDKEAEDLKTHLKEMVENTEAGKMAFHKVLDEYKNGAISRLDSMVSAAQDRLRQMQAIMAAMASMSGGATPS
SSSIPKYHTGGTAGLPPLANNEVPAILKLGEQVFTQDHIAKLKSIIMQPINFVGSILGNVRNAIPDSPIRNNQETAVPTY
NLSNFTIQANNPTELFSGLNNLIKSNRT
>lcl|NZ_CP026520.1_prot_WP_053228663.1_2821 [locus_tag=PC41400_RS14355] [protein=hypothetical protein] [protein_id=WP_053228663.1] [location=3186365..3186916] [gbkey=CDS]
MALIWILMIMACLLAVTVSIGLIISKSNKSKGILGQFQSKHVCGIPNLDSGVGADIKIYSDKLTINDTQIIPIDRYVQAR
CLLARELKLSEKQKSVILRAVTGGMLLGPIGAIVGGMSGLANQSKQDEILQPYIQVVYFDKNNEKKEALFLTGYSYDFTK
GIVRLFNESAGRETNVEELQYEI
>lcl|NZ_CP026520.1_prot_WP_042227722.1_2822 [locus_tag=PC41400_RS14360] [protein=site-specific integrase] [protein_id=WP_042227722.1] [location=complement(3187198..3188181)] [gbkey=CDS]
MAEVEKLPEVTEEEWNSVNEYNIEIVDEFLQQQHLSPQTLKQYESALKIFFRFVREKCKNAPLYELKPKHALQYQNFLMS
RGLSSSAVKLKRSAVSSLCGYLEVYYADEHPLFRNIYNKKIPNPPKTARHEKKPLSQAEIDNLIHVLEERSEWQMLAYVL
FSYDSGCRRGETRQLLKEVATYQYSKDMKTGEEKNYYLTHNIRTKGRGKEGKVRKLIFSDMSMTAIKKWLSVRGDDDCPY
VFVHKTKAGNTKQLSPTAFNDWCTDVFSEIVGRRVHPHQLRSSRATNLVAVEGKDIEKVKNLLGHQSSETTKIYVVKEGE
DDVDDLF
>lcl|NZ_CP026520.1_prot_WP_042227720.1_2823 [locus_tag=PC41400_RS14365] [protein=hypothetical protein] [protein_id=WP_042227720.1] [location=complement(3188194..3188619)] [gbkey=CDS]
MKEFNNLTDTLSFLQQQINSSLKEEVSDLTVKTLQENMQKEVYDVYQPKKYDRSGYQGGLIDPSNIEVKINGSLLSVESV
RNDEDRNVSEIVETGKGYTYNFEYAGKPRPFTDTTREELRESNKLIHSMKNGLHKRGIKTD
>lcl|NZ_CP026520.1_prot_WP_042227718.1_2824 [locus_tag=PC41400_RS14370] [protein=hypothetical protein] [protein_id=WP_042227718.1] [location=complement(3188603..3189136)] [gbkey=CDS]
MNRLTLTSDSINALDTDYDMLQKVDFKDGSYTEIYKSFRPTKIDEMVKSLANFITQYEKHQGKINDNKMFYSLLYLHIVI
YFSKLIPELPTDFPTKLDLFLKLLDNQYVVELEESFDVSEIAKVYSVIEKSLRIHKEATKQINAAREEVRNNLDNLELQN
SDIFENLKDTVSNERVQ
>lcl|NZ_CP026520.1_prot_WP_042227716.1_2825 [locus_tag=PC41400_RS14375] [protein=hypothetical protein] [protein_id=WP_042227716.1] [location=complement(3189301..3190035)] [gbkey=CDS]
MALIADVFIALLVDPSGQTFASTTLQSGDIDFNVKENEVRGGRGNALQGVLHSDRDISVKMTDSLFRYDWMARQLGQSIV
TGAGRAYAMPEWYTASGSSTIKISLPEAPIADTLVIYKADGTLVPKANYTLSNKDVTFTTGVTAGEKVDVRTYQYATSAK
TETINIDNSVFAKGTKLILETLEIDNDETPLYKIQYIFESAVPTGNFSIKTASERNGQAQQFDLKVLKPKDSNVIGQVLR
IPLT
>lcl|NZ_CP026520.1_prot_WP_042227714.1_2826 [locus_tag=PC41400_RS14380] [protein=hypothetical protein] [protein_id=WP_042227714.1] [location=complement(3190079..3190798)] [gbkey=CDS]
MESVKYILGKPELEKDAGFIYPVKVKDYDEFNDCRNVLYINKNHFSNEYQEYPLLDLIVYGYKDVEMIRQLKLLIKLVTG
HDPLLHEDNNGCSFVISADKKVYSGNYENFRRIVMRQNLLFEQKVYKNKLTQEWANKAIEAKSKNGIPITFEDMITTVSV
ITGKTYDQIGEQTIYQLTADFKRIARDKSYHTSIAMKCAGAEKVSIEHFAEEINMFENPYDNLFVGKENLNQLNKALRN
>lcl|NZ_CP026520.1_prot_WP_042227712.1_2827 [locus_tag=PC41400_RS14385] [protein=hypothetical protein] [protein_id=WP_042227712.1] [location=complement(3190798..3191286)] [gbkey=CDS]
MAKFEQLNQYVTNIFSVLIENQDLCKLLFYAVDDPLSEVDLTEDQRFELLHTHIYPMPKIPGEQSAQSSFLSIYFDNFKL
ANENKGIKDSSLVIDILIHNEIWNLHGTGLFRPYSILSEIDKMVNNERVAGIKKMEFDRGRLIRYNADYSGYQVTYSMSS
VN
>lcl|NZ_CP026520.1_prot_WP_042227710.1_2828 [locus_tag=PC41400_RS14390] [protein=Ig-like domain-containing protein] [protein_id=WP_042227710.1] [location=complement(3191305..3192531)] [gbkey=CDS]
MSWQERFALRMNAIGTTIRQREINNEILRIGEIFSDSPSYALVYQNDAATPADVWITDNSDSKGTKDIVSQLKVPLYAGD
LIEWNYEKWLITSMESISTIYYAGTMSKCLGNLKWLNGDGLMRDTWFTFNSISNANSGVNEGQIINLGNEYRHILIPLNE
NTKLISKDKRFILDQRAWKAIGVDRISKRGIINLTLKEDAVANSDNLELEVADYRGNTANYSLAIINDDPSSISVGFTLQ
LRSEVKNNGTLVSKPIIWSANNEAVATVNESGILSAHSVGQVIITASLADNPSISTSIDVTIVDIPKLYTIEIESNSTLP
HEIKNGQSKTYIAKVYENSSLVLENLEWQLFSDDQISPTSLGLINSQTNESCIVKNNRSNSGYIQLKVSLVKDASIYRWL
RLQMKPLV
>lcl|NZ_CP026520.1_prot_WP_042227708.1_2829 [locus_tag=PC41400_RS14395] [protein=hypothetical protein] [protein_id=WP_042227708.1] [location=complement(3192531..3192989)] [gbkey=CDS]
MPTPFSKVFSIFTGKIQDDLYSILSVEEAELDMIDLLNESIVKFEFPKEDIRNKDDLIKEFIEDLSLGTISILGNLMVLE
WQDRQINNISLIKQRMSSKDFKFTSQANHLEALLKLREETERRVFKLKRQYSYLQTGENYNKPDYSGLGGGT
>lcl|NZ_CP026520.1_prot_WP_042227706.1_2830 [locus_tag=PC41400_RS14400] [protein=hypothetical protein] [protein_id=WP_042227706.1] [location=complement(3193003..3193491)] [gbkey=CDS]
MANRKLTDKTMVSVFNNNGGVVFYYSELNRVKRRWDKPNVDKKISLEELKELVNTAGGYELLRDDLLITDIDVREELGLP
VEKEYMLDDQGIKELLCRSQEDLEEVLSNASDAIKEKIAHVAILIQLADLNKIEVIKANTGIDILSAIQQGKEDQKTGVK
TK
>lcl|NZ_CP026520.1_prot_WP_042227704.1_2831 [locus_tag=PC41400_RS14405] [protein=hypothetical protein] [protein_id=WP_042227704.1] [location=complement(3193560..3194552)] [gbkey=CDS]
MADMTIKQLAVAVYKKQAPTNFSLNDMEGALRDKFRELASDFNTYRRNKIEIFELIEQVIDEVAPKKIIEAIGMFAEVKT
YGQGERARFKRKLGRNNVRRFITRVGLGGIYERVRLDSDYVDVATHAYGGAAYVEFEQFLDGTMDFTDLVNLIIDGIEYK
ILEEVQAALIATFNGLPAANKASGAAFDAAEMRRLITTIKAYGGNANIICTPEFAGTITPASGFVGDPDKADMRNQGYIG
RFEGANVIVLPQSFTDSSNTTKVLNPQFAFIVPTGGTEDEKIVKVALEGQTVVKEVQNADDSMEFQAYKKIGSNILFTNY
FAVYKNTALQ
>lcl|NZ_CP026520.1_prot_WP_042227702.1_2832 [locus_tag=PC41400_RS14410] [protein=hypothetical protein] [protein_id=WP_042227702.1] [location=complement(3194590..3195126)] [gbkey=CDS]
MAIVKQPSVYGVVELNHIAAVKTGEIKAQFPLNTTDFASTPAENGMLLIVDEVIEDIKKPTGITNYVYLHASVEKDYEGK
GRKHFSVKPGEFLPKLLKLKRGDTFETNSFEYDTATYANYVAITAAINSTTVYGIPSTSGYIRIVASPGGTEAVTLKVVK
GVTLPNGELGLKFVVDKG
>lcl|NZ_CP026520.1_prot_WP_053228662.1_2833 [locus_tag=PC41400_RS14415] [protein=hypothetical protein] [protein_id=WP_053228662.1] [location=complement(3195148..3196788)] [gbkey=CDS]
MNLTVSTSLPITFEQIEKLPDSRFTKVKIYLCHTGKNLNNSYFEKSVIEEAIPSLANIPILGFIQVDNLNEADFKGHEQR
LMIDENGVTVEYLGRAYGVIPEDNRAYFENKLCDDNVTREFLVVEGVLWNKFKDATEIIINNDSRPQSMELAPSSIEGNF
KKDGVFYFSKFQFEGACILGSNITPAMRGSVIENVDFSLNSASLQSMLNEFNTTYSELNQPSKEDDNKYKGGVNLNEKLE
LFKKFPTLKDEDIAELKANIEQYSLEELETNLNQIVKSQTDSNTFSLTAQQLQQEIRGALTLERVIDKWNDSVRAYWYID
HDDNRVYAEDTQDGYLPVGLNYTKSGDSVVIDFESKKRIKWVPQDLEEGIETAVSFVSIDRSAAELEITKKTVEQDFFTK
LEEINTELDELKPQFEATKAQNEKLQAEVSEYSSNVDNEKLKVQINELSAYKLAREKQDKLDVIKNFKDLTDDEKKPFQD
EVDKYAIEELEDKLFSFVGRKGLSSFSANRGSKEYLTTSLANFEAQTKSNEPTWASLVEDFKNKSN
>lcl|NZ_CP026520.1_prot_WP_042227700.1_2834 [locus_tag=PC41400_RS14420] [protein=hypothetical protein] [protein_id=WP_042227700.1] [location=complement(3196806..3197879)] [gbkey=CDS]
MDGVFYGYERQLDDVITLQQLPPQFCRSKYKINGSYAIEFNLRFFDTYKDIQLKLKLFEMFPEDFLQLYLDYKDGKSTEW
VQLNPSYARCHKFQDDARPMLADIFPELINLKEYKELDKSKTKMDLYKLIVQKLPVDDDGEPTILLEEAKALHSNAKKMI
SQEGIDVLTTPLEVQGVNLQERGSTINDNINRANNMVFDEAGASRIIFNGGSDGGSIGLSYSINVDENTMFPLLDQFKRW
YDNKFKSLVNTKNYWFENLFPEVTRFNWKEKYDQYKDAATLGYSKLLPIVTMGIKQTTFLNLLQYENDLLGLNDLMSPLR
STHTQSGEQGRPEQGEKSLSEKGIETRNKNGNKNRSK
>lcl|NZ_CP026520.1_prot_WP_053228661.1_2835 [locus_tag=PC41400_RS14425] [protein=hypothetical protein] [protein_id=WP_053228661.1] [location=complement(3198271..3200070)] [gbkey=CDS]
MINKNMSQRKIDGYLKLVEVIQWGRRYPVRFVERFFGIELLDYQKYIFMQSWSTPFVLWVMSRNAGKTTLGSPFIMAKSL
LVPNLQVYILAGVGSQSQEMFMKIEKIAKREIASFTGLTDIFYNETIKSAANTDGFTHNPASFQYKLYNNSAVNSLNGAF
DNNRSKRSNLNFYDESGFAPDDLFLTSEPFTVQNSDFRLGGEVDVTMLPKQFPNQLIYASSASSTDTYFFSKYRDFAQKM
FLGDKRYFVADISCDLVINATHNGKKYPVSLLSQETVDAAMRDNKEKAMREYKNVFSTEGGNDQIVKRAAIIRNSELRPP
TLANEGGSRYIIAYDPARSYDNSVAIVGEIICDEQIGYKLKICNAVSFSDMAKKKKTPMRTPEQTEFIKQMILDYNGAGF
ADYENIEALLIDSGAGGGGTIISDYFMEQWLDKGGRLHKGLIDKIESADYINKFPSAVDKIKLLSPKKYKKDMFEALIEM
MNLDLISFTDTYDNKGIILTNRLDDKGNVTESKYKLSNDEITSLIQLDLAKEELVSIYRFESTNGNCRYDLPPDKQSKMH
DDRAYCVAMLAWYLQQLRRESITNKSRPKSDISRYVFYN
>lcl|NZ_CP026520.1_prot_WP_042227697.1_2836 [locus_tag=PC41400_RS14430] [protein=hypothetical protein] [protein_id=WP_042227697.1] [location=complement(3200060..3201130)] [gbkey=CDS]
MPKLKNNAPTIEESKDKTCTSCGKTKKMRIDFYMSNSKHHSDRRYPVCKVCIKKDLNINEQSSIRSVLLEMNRPFLFNVW
SNALIEAEKKNSDPFGLYLKSIQLSYKELTWPDSDVEYVGPLIENKINPTATSDQINILINTNDKRNEEDVVTMLGYDPF
VVETAADRIFLFGRLVDFLDESTLEDGFKLPAVVEIVKTFHQIDKINAQLSSIMTDSKSVASNIGGITSLVNAKEKMLKS
VLALAKDNGISVNHNNNKSKGSGTLSGIIKTLHENGFDEAQVNLFDIETSEGMLQVADISNRSILNQLQFDENDYVSMLT
EQRELIQELDKKVLALEEENRLLKRNEKKQGLGNDK
>lcl|NZ_CP026520.1_prot_WP_042227696.1_2837 [locus_tag=PC41400_RS14435] [protein=hypothetical protein] [protein_id=WP_042227696.1] [location=complement(3201295..3201528)] [gbkey=CDS]
MKNDISPINVIVLSDFEKESDCMIAACLDILETCKSRIDNESVLRMVAQEIREITLREILTKEIQMKAAILEQTKSW
>lcl|NZ_CP026520.1_prot_WP_042227694.1_2838 [locus_tag=PC41400_RS14440] [protein=hypothetical protein] [protein_id=WP_042227694.1] [location=complement(3201518..3202723)] [gbkey=CDS]
MDQPETFRKPGEDLDDYLVRLGDNFDLYGLTWTTASSLLNSEAEDEYSESKWRKDYNCYLRWKDKLTLKLSNDGEYLQEI
HDKTTELLKEKIRLQDQKREYTTTVRYQARFEHLKNEIHKSILALAKIKPLNFQSPAPVFSDVRANILWSDWHYGSDFKN
SLNEYNPDIFRARLQRLVSQTIKYIKLHKVNTLTVGALGDFLAGAIHISNRVQSSETIIEQIQVVSEALAESLAELSKHV
QQICFINIIGNHARLIPNKTESILNENLENLIPWYLHSRLKDFKNVEICQDTDGYYVDNSFSPSHVYVHGDLDHVSSVAK
TIPQVIGVVPRYIFCGHIHHDTVKEHGRTKVISNGSLIGVDDYAVSKRLYAEPMQKMHIFNGDRIEYTIDIPLNRCDVNE
K
>lcl|NZ_CP026520.1_prot_WP_042227693.1_2839 [locus_tag=PC41400_RS14445] [protein=YonK family protein] [protein_id=WP_042227693.1] [location=complement(3202727..3202924)] [gbkey=CDS]
MSKKNHDKSSFNRTGLFDYEKMTINYEDKNGVYTFNLKSALEKLDGQNISLSVEAETPAAIVDEE
>lcl|NZ_CP026520.1_prot_WP_275938056.1_2840 [locus_tag=PC41400_RS14450] [protein=HU family DNA-binding protein] [protein_id=WP_275938056.1] [location=complement(3203020..3203301)] [gbkey=CDS]
MMNKDQLVKQVANDTNLSNKDVDLTINHALEVIKQALSNGDKVTLHGFGSFENRTRAARKGRNPQTGEEMLIPESKRPAF
SAGKKFKDALNCK
>lcl|NZ_CP026520.1_prot_WP_241688868.1_2841 [locus_tag=PC41400_RS14455] [protein=RNA dependent RNA polymerase] [protein_id=WP_241688868.1] [location=complement(3203552..3205909)] [gbkey=CDS]
MSKNTNARKLREDHLISSKLVTAFESTLTRTINAQINCICADLIIVRTYYLKVLENLITQGFIFKNEKYVYLTASAGQIR
TKKTVFIKESLLQKHRDTLTCGLTVEKINSLGGVNINKYLAYLALCFSASDPWKDFNIDRAIVVDDMETNVRSVVDFVDD
KSYEITRQQMDIPINHTDGCGMVLPRKNKKSMMVRLPWVKGLLVPFPFDKFIREKSKSSKVKDIYGKEYDIIKDRIEVIF
TKSQFKMWKYYSSWDEYRAFYKLHGCHASKCNEEDDVFRKAKLNYQMLQTLTDFSDNELMSVSAKTIEKINNIGRDRDTM
LSILGVKESNNKDYFQQALELYPELLNDTYSREVLKQVKKSIVKEAKSGKIDVEGYYTFVSPDLYAFCEYLFLNDLTPAG
LLDNGQVYCSLFKEYSTLDCLRSPHLYREHAVRQNIVDKEKSRWFVTKGVYTSCHDPISKLLQFDVDGDKSLVCADSVII
DVAKRNMKEIVPLYYNMSKAEAQMITDAEVYNGLAAAYTGGNIGLYSNNITKIWNSENPNLDVIKLLCCENNFVIDYAKT
LYKPTRPDSMAKLIKGYTKAKNPNFFIYAKDKSKDSVENANASVVNRLQKLIPNPRLNFRAANLGQFTYKTLLSDTNKYI
DVSKENNLIEKFKELDLKRRFFSLVDEEDLYSKSYRYKEIRDILLDYNTNINYIVDVLVEYLYVYKKSSYKTTLWSSFGD
VIVENIKRNVTHPLKGGYILCNVCGDRILETSNRRKYCERCWKEEQKRIKRESWHKHKSKYRKTC
>lcl|NZ_CP026520.1_prot_WP_129112484.1_2842 [locus_tag=PC41400_RS14460] [protein=hypothetical protein] [protein_id=WP_129112484.1] [location=complement(3206112..3206294)] [gbkey=CDS]
MRMIGASFASWQDNYYDGVENDEFEYRGLEKEMYDEYINDTSDDYLLALKRQGGKNNQSR
>lcl|NZ_CP026520.1_prot_WP_042227687.1_2843 [locus_tag=PC41400_RS14465] [protein=site-specific integrase] [protein_id=WP_042227687.1] [location=3206456..3207499] [gbkey=CDS]
MNKATNNIVSLHHDSVYEDIQAFLSKHEIQSASTAQNYERGIRKFFMYMKNKNIEDLKREDLQFRNVDMIRYQKYLIDQY
NKKTTVNAYIAAVLSLFEHLERNEYPIKSNILRLDFLPADSQSYGKLTLDEARLFAELALSQDKGLEKSVLIRLAYTTSI
REGALLSLKEDNIRQSECGEYYIVEVIDKGGVKDKKPIRPDLYKALMDLRQQHYCKRYTDNKFFHITPKTISAMMKKIRN
QLGIPEHRNIVFHSLKTVAINWELETTGDIHRAKRQGNHKNVNTTMIYANNEIGMENMAGIQMENDIKEDIFEELSREEL
ISLLKGIKNGLGYQVRSAARNMIANRA
>lcl|NZ_CP026520.1_prot_WP_129112485.1_2844 [locus_tag=PC41400_RS14470] [protein=hypothetical protein] [protein_id=WP_129112485.1] [location=3207515..3208261] [gbkey=CDS]
MAMRRRELVYIHNEIFDDLKNFDGINTGNIATVYCYYCLLCYLYRYCKYPIPAEKIKQMLGYSSSYTNINKIIKRNGLTD
RIGLTHSTRDFPLQYKYETGELSFTTYFSLDAETRRLVDMSKNYFIKQPLKAFYRTEESKQIGIKDGTYFVIDNTHVFDF
VCFQTILQDEIGLNGFYIYQYISHKCDLFLNGFDIGITQLEKSISIPKRTIFRYLELLQSKGYITINQSNRRGEATKANV
YSISGRKS
>lcl|NZ_CP026520.1_prot_WP_042227684.1_2845 [locus_tag=PC41400_RS14475] [protein=hypothetical protein] [protein_id=WP_042227684.1] [location=3209487..3209849] [gbkey=CDS]
MIKRLSKTIITVLLCLSVLTLSACGMNSEFVKDTKAVMESIAYDVMHNVTDTESTQKKVDAYHAKWKEKVKEGDETNLLN
IVGELMGYEILYALKTDGADRMVVFTDYRRKENEAKKLME
>lcl|NZ_CP026520.1_prot_WP_042227683.1_2846 [locus_tag=PC41400_RS14480] [protein=DUF4258 domain-containing protein] [protein_id=WP_042227683.1] [location=3210003..3210299] [gbkey=CDS]
MVSAVLTKIQSLATQGNYFMTKHARIQMIQRGLTDADVRDILTNPNQILRTDTNNPDGVTSYKIKGGQHDHRLAIKFVHN
PDELIVIITVMDNQVRAN
>lcl|NZ_CP026520.1_prot_WP_042227682.1_2847 [locus_tag=PC41400_RS14485] [protein=helix-turn-helix domain-containing protein] [protein_id=WP_042227682.1] [location=3210342..3210779] [gbkey=CDS]
MELKVCVECGSKQLSPIMDTIEQTVEGKTLVIENIPAVQCDSCKEIYHSSKASQYIDKQIEVFKAEGFENKLREVTKEKG
LTQEQLGQLLGGLTKQRVSQIFKDNNLDVKTMYKLSKVIDEPIENLFTFNRVVEKDNRFYIVHKD
>lcl|NZ_CP026520.1_prot_WP_042227680.1_2848 [locus_tag=PC41400_RS14490] [protein=hypothetical protein] [protein_id=WP_042227680.1] [location=3211029..3211454] [gbkey=CDS]
MSNAFVIAKFSNSKGKTEIYYGATDPIVSKLDSVFATYEFFPDEKAIINRLSEISSHWGNRGFIYTNINNNKEKNMYWIL
SIFKKNTREYYYGERSHVIDKATSVKGKNRLIGKAYNTKELMLWQLEYHKKNSPKSKYIAV
>lcl|NZ_CP026520.1_prot_WP_042227679.1_2849 [locus_tag=PC41400_RS14495] [protein=hypothetical protein] [protein_id=WP_042227679.1] [location=3211508..3211897] [gbkey=CDS]
MFNFAVTEKVFESKTQFKKAIQAVKNKVLNTEDVILLINANIYNLYNSLVSHFFGGGPVNQENIYKQNLVIEYIEEYEPR
NKHLFYILIENGLIGVCGRGFTFESDRINQPQSEYERMIEIIFNKGFVF
>lcl|NZ_CP026520.1_prot_WP_042227677.1_2850 [locus_tag=PC41400_RS14500] [protein=hypothetical protein] [protein_id=WP_042227677.1] [location=3211979..3212425] [gbkey=CDS]
MNGINKVARINIYNRTEEEYKTLQQLIDLDIKENNIKSYRRDSGFVAPYWGNYCMVGLPYEKALEYAEKYDLTVDLIQTI
MQREVIQSLNQGDKVRYTVDTIFGEMSGKGTVHFADDNSVTIRIYKSKTKAHIIKVGDFATIKKGWKE
>lcl|NZ_CP026520.1_prot_WP_042227674.1_2851 [locus_tag=PC41400_RS14505] [protein=DUF3560 domain-containing protein] [protein_id=WP_042227674.1] [location=3212479..3213654] [gbkey=CDS]
MSNQYILNRETQKIELRFTKDEYKSMPEVQKKELRRFFTWSSYAGAWVSKSTNNHYSALRVAEQLGFTNGGQEGERLSYA
EQLERKAEKAERRAERYEEYADNAANRAKGLQQEFDELRKDWSWLTQPNINSSAGRAFTNQRNRVLNRYHKGFDEYRKSE
YFKERAETAHATADQTKLNSKPYLNNRIEECQKEIRSLERYIVKAEERNDEEWLQKLLDRMEYEVDKLAYLQNRMDEIGG
FEYSKDTVKVGYLIKVRGLWATVLKVNPKKVSGDYIDQHLKGCYCYYPYAEIEEMKIPEGWTDKKETEPNPFNVGEILVL
KWSTSDNIRAAFQIVKTTDKSVLTQEIKIEDLKPIADEFISEKQQRRLVKKTRDGISAVSHGNCYLYRYTR
>lcl|NZ_CP026520.1_prot_WP_042227673.1_2852 [locus_tag=PC41400_RS14510] [protein=hypothetical protein] [protein_id=WP_042227673.1] [location=3213718..3213924] [gbkey=CDS]
MNQELNIPIRFDGRVALNNKNRKWEIKALRFYPLTGSLCEAGGWVINVLNSRKNAKEWAEENGYTVIR
>lcl|NZ_CP026520.1_prot_WP_255246829.1_2853 [locus_tag=PC41400_RS30590] [protein=hypothetical protein] [protein_id=WP_255246829.1] [location=3213936..3214070] [gbkey=CDS]
MKTKEDEIIKQFEQLIKEETDKKTKELLKMTLASYVAADSFNTK
>lcl|NZ_CP026520.1_prot_WP_053228659.1_2854 [locus_tag=PC41400_RS14515] [protein=methyltransferase] [protein_id=WP_053228659.1] [location=3214163..3216784] [gbkey=CDS]
MWYGDKREAKVLIMRELVKRGWKTYGYKEDCSDSMTDYFDPASWDGLAEKNGYVLVIDHSNTYYSGYEKKQVLYSSVKTN
QDINRKLEKLHKLAAANSGATEGERAAALAGIERLRCRDTEPQFTVVETYPSFTYENPGNCSWHIEKDGQIIGKGTGVFA
CNDYDYKDKTKTAEQQKQERVIAIVNRFEKVLRNSDTLESEIILVPRTIVKFVEVEKTFITAEDFKTGTTFIMKCDYTNG
NWKGTKWQCVYVYDDKKSGSFSKLGKNGKPSKAKGKTMFVSTDKINDLLQKGHISIVELQEATEYEEKIVFKKTSRKQPK
ANMDAPALSGDTEEASTEKKQTYTKNETVTDEKQATKRQLWALHCATKLNTTNLVITKDKASQLITKSKNGHDITAELKA
LLGFAEAQESPITAGAHDTSITIEDAADWIMDYSTSIMSDQYGNIKRPETEEEKTEYKAKISDYVKERGFFVSEILIGLF
RSYGQGSSLIVEALESLKVMNISSSKMKQQNGEKTDKNPISSNQEQNIGKIEKQIESAQKKLNAISRNYLTNTWKRQNEE
ENRENKRERLRLEIALLQYLKDKAIHNSMTQLEQALLVGTFRDDIHNRFISRTKFKRETKYPEIDYSLPLDGMWNLQVPK
KQKMLNRANICNTYQYNSALDEYETIIAEIEKPVNPIIKKIKKLEAEVKFKKIDGYYPTPKTIVEHMLSYADIKDGETVL
EPSAGNGNILDSITAHCQQNGIAADLEGIEWNSSLREILELKQYNIVGYDFLEHVRYNHFDKIIMNPPFERNKDIAHVQH
AYKCLKNGGRIVAIMSPHFTFANDFKSVEFREWLNERGYYDMLPEGSFKESGTSVNTVLVIIDKMEDDAAEAI
>lcl|NZ_CP026520.1_prot_WP_042227672.1_2855 [locus_tag=PC41400_RS14520] [protein=hypothetical protein] [protein_id=WP_042227672.1] [location=3216946..3217239] [gbkey=CDS]
MLEIDYKPKFYKRKLKGLIQESRGSKFQFEDTLDGGASYILVDQSVEGINNRILDVINGTRDKAYLSHGMRNIDLEKLSN
VKWQIWDEFDIYEFEMK
>lcl|NZ_CP026520.1_prot_WP_042227670.1_2856 [locus_tag=PC41400_RS14525] [protein=FRG domain-containing protein] [protein_id=WP_042227670.1] [location=3217294..3218082] [gbkey=CDS]
MKFLQTELFGEVFAPESFLELTELVIINNSGRYNNVRLWRGQSNIDWPIHSGAYRKLINSKPNILENDIINYEESLIKQA
RHKGYGKLYGNEISDSELLAKLQHHGAATRLVDFSKNSLVGLWFCVQANQDKTGLLLGVHTNFVGGGIEGVLDDRDYSDL
TKDLSKYDHPMFIETPVVSKRIAAQHGVFLYSDVSGIKSGSLKLPKENNANLSIAITAKLKLECRKILIDCFDIRNETLF
PDLDGFSNANSTNSDSNDMFRW
>lcl|NZ_CP026520.1_prot_WP_129112486.1_2857 [locus_tag=PC41400_RS14530] [protein=hypothetical protein] [protein_id=WP_129112486.1] [location=3218319..3219143] [gbkey=CDS]
MSKLSKEEADEKTSVYNSDVKASFGQKKILFDNLMSYAKIVIENHEQRQIKCDQIEMPERHPIFDTIKLIGLRLQTKYIT
NLLYQGKEFERLPNIDAWHVLFSISEVIDKSNGKKMYDFFMDLPAENERVVLKEDLVLPWPWNKGRQLNSMVNIGKGRFW
GEWEEKSNNHNIEFWLPMRIGFVNGGNHSISAGILNGVGDLKPAYVYDISPIYEHVYTDGINFYRTNGNTIIAKALSVEF
AALFEIGRLMKDKNITKRNEIVLSQTNDEVSRIW
>lcl|NZ_CP026520.1_prot_WP_042227669.1_2858 [locus_tag=PC41400_RS14535] [protein=hypothetical protein] [protein_id=WP_042227669.1] [location=3219306..3219488] [gbkey=CDS]
MDFTPGNIYSLDYGVVIKLATFSRKEGEYNFFFDKDGEFALSDSFLSKGIISIKLMNDEL
>lcl|NZ_CP026520.1_prot_WP_042227668.1_2859 [locus_tag=PC41400_RS14540] [protein=hypothetical protein] [protein_id=WP_042227668.1] [location=3219530..3219808] [gbkey=CDS]
MSSYHTAINKYGNLDVIDESRSVVISLRDATKIDTRKSQTKIFDDADDLIEFINERTAAADKMNSVYFSPRAYEIIDEHY
IKLYHEMVVSFA
>lcl|NZ_CP026520.1_prot_WP_241688869.1_2860 [locus_tag=PC41400_RS30305] [protein=DUF262 domain-containing protein] [protein_id=WP_241688869.1] [location=3220057..3220947] [gbkey=CDS]
MLIHSSLTLFVPDVYAVQTPKIINEKQVNVYDVIDGQQRLTIFYSYRKDEFALTKLEPVTLESNGETYDISGLRFSELPE
EVQDSINSFMITFRVLEFDENDDEESIIDTVFYRLNNGKPVSKEHLAFVSASKKVQNYGHERINKSGLFLNVAHFPEGSI
KKSDRQMCILQSIIIAGKLDFPSFSTKDVEAIFKQNEIEDETLQLLDSAYNNIEKTFQDKSKFVTKVNIPSMVALFLNNS
DQEKVSAFLQWYSQNNKKGDSYRRYCVGGTTKKDQVWGRVNGLQALYNHFVSESSE
>lcl|NZ_CP026520.1_prot_WP_053228657.1_2861 [locus_tag=PC41400_RS14550] [protein=helix-turn-helix transcriptional regulator] [protein_id=WP_053228657.1] [location=complement(3221189..3221560)] [gbkey=CDS]
MNRFKNEESYLLLLGARIKQIRNDKGITQEELAERMNTSSPYIGNVERGQKNITLSSLFRFADALNVRVMDLFAFEAFSD
TGIVKPSELAITEDKMLPLIELLSQCNTADRDKIYRMLIEFLK
>lcl|NZ_CP026520.1_prot_WP_042227666.1_2862 [locus_tag=PC41400_RS14555] [protein=DNA ligase] [protein_id=WP_042227666.1] [location=3221889..3222722] [gbkey=CDS]
MFVSPMLLQTANNPSDEAAYINELKFDGFRLIISTINGVKLYTRHNNEVTSIFSELLDCPLPEGTIIDGEVVIIDAQGKP
DFEALSKRFKSKTDKTLVTFFAFDIIRYKGIDTSGLPLLKRKELLSESFSETESYKQVKWSLGSASALYELVEANALEGL
VQKMAESRYEIGKRSWNWQKVINWTYAEVFITGYKKNEFGWLTSVIDETGKMRAAGVIELGVNPEAKKAFNVIKRSITYK
EDKSTVYLEPLIKARIKTRNWTKKGLLRSPVFVEYVL
>lcl|NZ_CP026520.1_prot_WP_042227663.1_2863 [locus_tag=PC41400_RS14560] [protein=hypothetical protein] [protein_id=WP_042227663.1] [location=complement(3222751..3223227)] [gbkey=CDS]
MRKDNWTSQEDKILADTILAHIASGSTQLQAFEEVSAKVDRSSAACGFRWNSELRKIYTEEIKKAKTARAESKMKRSNND
YSEQVEQDYEDTFGGLLEKLKSKISLLEKELDRVKNENTRLKITLQDSRPDVVLSEDFKNMIKIIERARSLGVLEKTS
>lcl|NZ_CP026520.1_prot_WP_042228817.1_2864 [locus_tag=PC41400_RS14565] [protein=type II toxin-antitoxin system PemK/MazF family toxin] [protein_id=WP_042228817.1] [location=3223498..3223860] [gbkey=CDS]
MTQKVEVNRGEVWTVRLDGNVGSEQGGIRPCVIISNQKGNKFGTTVIVAAISSQIQKAKLPTHVVLEGGVDGVLLDSVVM
LEQVRTVDKSRLINKVAELSSDVMQKVTQAHKISCADMFN
>lcl|NZ_CP026520.1_prot_WP_129112488.1_2865 [locus_tag=PC41400_RS14570] [protein=hypothetical protein] [protein_id=WP_129112488.1] [location=3224077..3224289] [gbkey=CDS]
MQDIRDFLAKRITYAKRRAFTYKNEHGENPEQTHNYFGGYSLGYWEGKVAAYQYILDELEGVGDVENKRT
>lcl|NZ_CP026520.1_prot_WP_156972796.1_2866 [locus_tag=PC41400_RS29785] [protein=hypothetical protein] [protein_id=WP_156972796.1] [location=3224270..3224419] [gbkey=CDS]
MKIKELEKLIDDRHYRNDPVITGMLALIKSQSKEIEYLKKKTEQIIRVF
>lcl|NZ_CP026520.1_prot_WP_042227661.1_2867 [locus_tag=PC41400_RS14575] [protein=hypothetical protein] [protein_id=WP_042227661.1] [location=3224474..3224737] [gbkey=CDS]
MNGLRLACNLYGKTYSDIANSIGINRANISIWLKTGVIPEKRISQLKKMFPEFTYEDFFKELSEDEIIAIKKSHICRLVN
EYGINRH
>lcl|NZ_CP026520.1_prot_WP_042227659.1_2868 [locus_tag=PC41400_RS14580] [protein=hypothetical protein] [protein_id=WP_042227659.1] [location=3224864..3225379] [gbkey=CDS]
MVDEKTLFYAHTENLKRELNVDPLVFFGEEAKEKTKKYGSLFLGAHVWEENPEFVAVADTNNIGILAHEMRHSWQYKNRN
KENYTFRSSSNKPWILKKLARALYLIYYSFNRKEMDANNYAIEYCRKTGLHHEVLRIEKLIKENRLAQVITILAHPLFVY
LLYVVWAYLIV
>lcl|NZ_CP026520.1_prot_WP_042227657.1_2869 [locus_tag=PC41400_RS14585] [protein=hypothetical protein] [protein_id=WP_042227657.1] [location=3225501..3225689] [gbkey=CDS]
MIWHVGYDGAAMPVEALTKQEAENKAKTYLDVKGTKCNLIYAQEASSGYLEWINDFGINVIR
>lcl|NZ_CP026520.1_prot_WP_042227656.1_2870 [locus_tag=PC41400_RS14590] [protein=hypothetical protein] [protein_id=WP_042227656.1] [location=3225730..3226074] [gbkey=CDS]
MKIYKLVWRFDKEEYVIGTNIANACNEAGIGNGALIALDYWEEVNELPDQLKRFEIVIDDKSDPDIIHNPNFIEQLKSSG
FQKGAVTEINLANRKFCFEYVGNSPTGYQYRKVI
>lcl|NZ_CP026520.1_prot_WP_042227653.1_2871 [locus_tag=PC41400_RS14595] [protein=hypothetical protein] [protein_id=WP_042227653.1] [location=3226125..3226364] [gbkey=CDS]
MFKKLEFKSTSIHKHESWVDFGIKKSHYLISKGNNDTYMIWFDGKDIGRDVHTLEKSKELAQAHYDNSCRLIVEAITMK
>lcl|NZ_CP026520.1_prot_WP_042227651.1_2872 [locus_tag=PC41400_RS14600] [protein=hypothetical protein] [protein_id=WP_042227651.1] [location=3226384..3226608] [gbkey=CDS]
MDQLIEDLTVVLIHLTASGNESWRGYDTDVLRKLKKENLIGLSNSWQTVYIGENFKNKAIEIINKYQLVPEGKE
>lcl|NZ_CP026520.1_prot_WP_042227650.1_2873 [locus_tag=PC41400_RS14605] [protein=hypothetical protein] [protein_id=WP_042227650.1] [location=3226605..3227168] [gbkey=CDS]
MIKVNLKALDFTPHAIQRAKERFKMLNEADIINMCKSIIRAGTYIGEITCDKGNRSHLFASGKKGIILSQDLTTVITVIE
YKRGHIPDLDKKNPLHDKLVKLYQSEIRRLTKLENTLYKKIKETKLYNDIELANLNLKLYKTRSIKVKKEIEQRMAEIKI
EFEEQELVMKQIESDKRIIAKSMACLT
>lcl|NZ_CP026520.1_prot_WP_042227648.1_2874 [locus_tag=PC41400_RS14610] [protein=hypothetical protein] [protein_id=WP_042227648.1] [location=3227209..3227472] [gbkey=CDS]
MYGKWVTLNLGEDFQCKVWVSCLPSTLESELQRMAINKLQCNLPKEKKGFSAAMDSLEIILDRARAEDFDSKQSKCLEQR
LRLGEHK
>lcl|NZ_CP026520.1_prot_WP_042227647.1_2875 [locus_tag=PC41400_RS14615] [protein=hypothetical protein] [protein_id=WP_042227647.1] [location=3227506..3227862] [gbkey=CDS]
MGKLMPIRIIQHYDEKIIGHIFYDELQYCIECSLHFSCNIVDSNGNGIKCPKCSSGDHLLDVGTVEIKLVETSNPSNIAS
EWAAAYEDEELEPEYHYFGVKPQIINNTNCYPWDGSEL
>lcl|NZ_CP026520.1_prot_WP_042227646.1_2876 [locus_tag=PC41400_RS14620] [protein=hypothetical protein] [protein_id=WP_042227646.1] [location=3227934..3228239] [gbkey=CDS]
MQTPSIFQLGKLSNLLDEINKGTGSVWSFVSCGSDSAMTVSLTFPNLTSDLESINNSKNKITEYFGLDESDKRVSISSGY
GKTELSIILESQQIIIKVTNK
>lcl|NZ_CP026520.1_prot_WP_129112489.1_2877 [locus_tag=PC41400_RS14625] [protein=hypothetical protein] [protein_id=WP_129112489.1] [location=3228312..3228617] [gbkey=CDS]
MGNIPEVICSESLTVDEYKIIMSKIWDSHRFGRLSKNRNRICRHIKYVCPNWDLRDGRCFSIGFDDVLFDFRGSYRSMFE
RIIGWLEGEEPECGGMLVQPK
>lcl|NZ_CP026520.1_prot_WP_042227645.1_2878 [locus_tag=PC41400_RS14630] [protein=hypothetical protein] [protein_id=WP_042227645.1] [location=3228657..3228860] [gbkey=CDS]
MIKYHESIIKRELYKYDDVEERERHHRQMFENGWHCKSQFDFLENYDWSNGTPAAEYLKQEQMDDKK
>lcl|NZ_CP026520.1_prot_WP_042227644.1_2879 [locus_tag=PC41400_RS14635] [protein=ClpX C4-type zinc finger protein] [protein_id=WP_042227644.1] [location=3228884..3229108] [gbkey=CDS]
MSESNIGVKAMHEIMRKAKKYDELLVFPSIENELQCDFCGKFQSELNKMIAARRVVICNECVEVCNQVLEEDNS
>lcl|NZ_CP026520.1_prot_WP_042227642.1_2880 [locus_tag=PC41400_RS14640] [protein=hypothetical protein] [protein_id=WP_042227642.1] [location=3229138..3229431] [gbkey=CDS]
MIKRRFLEYEYFKNLLVKQNDGLTEEEAVKVLNEKDLIWGDLTFTFKQQGNGWRTDIYCNNSDTYITPRELNPVEEKWFL
DQVTELSMQKYLALQTA
>lcl|NZ_CP026520.1_prot_WP_241688870.1_2881 [locus_tag=PC41400_RS30310] [protein=helix-turn-helix transcriptional regulator] [protein_id=WP_241688870.1] [location=3230143..3230457] [gbkey=CDS]
MAPKKEEISLRLKKSRELKGLSQREVSQLSGINNKTISGYENNVSEPDLLTISKLASIYGVSIDWIINGLHFKIEGSNGV
LEEQLEHSRELLLKFKELAEQIKV
>lcl|NZ_CP026520.1_prot_WP_042227640.1_2882 [locus_tag=PC41400_RS14650] [protein=hypothetical protein] [protein_id=WP_042227640.1] [location=3230483..3231241] [gbkey=CDS]
MFLEDIIKFANENEIIYFINELDPDEIERVNISISSSKENFLKLFLLAKRGNYPVFITESKLTMDEVQSSLISYEDLIKN
HLYKTSGNEVKLKITNIVDEYNKVIEDIPVDVTYKVSVVVILSTYYMSIDHFLNDEGAIWDSQLFYSLDELIEEKFGIEE
INKLEQNRVIYTEEITQKVQGKWIEHLRNDDEFRGCTNRDLRLAYLSKQWLRFKQTLSDEEKEINSVHPQIVMEQVWKEI
KESISISRIKNK
>lcl|NZ_CP026520.1_prot_WP_053228656.1_2883 [locus_tag=PC41400_RS14655] [protein=lecithin retinol acyltransferase family protein] [protein_id=WP_053228656.1] [location=3231321..3231794] [gbkey=CDS]
MKYTLTLLSEAIERVTRTTINAVESSQAAPYKAIHAFFGKDPEIVKGDHISVFCGYIHHGIYIGDGMVIHFTKVDGLGII
QMDTIETFKNGKAIEIRNSPLNCSREEAIQKAYSKLYTTGYNLVFNNCEHFVVWCRDGDNKKIYTPSSAAWQVMGNR
>lcl|NZ_CP026520.1_prot_WP_042227639.1_2884 [locus_tag=PC41400_RS14660] [protein=hypothetical protein] [protein_id=WP_042227639.1] [location=3231893..3232087] [gbkey=CDS]
MSRVRYDFDSVLSNAKTFSDGVMADEKIREMTRESIIQMVVSSCFEYRDNEEFRKRVDEYVENK
>lcl|NZ_CP026520.1_prot_WP_042227638.1_2885 [locus_tag=PC41400_RS14665] [protein=hypothetical protein] [protein_id=WP_042227638.1] [location=3232375..3232866] [gbkey=CDS]
MNVDLSHFTVANIWSIFGLIMDLVGVIFLGFAFFSKSFDDLRKESGTFFNYNPSLFLNLLDQKASGVAGTVALSLGFLQQ
FIVNIPISTSIPSLVLVAVLLFFNILIVVGLLVCKKYYVLYQLAKIALFWDKGKFQDDVQVKGRAFQEMIIIGKVKMQEI
QRR
>lcl|NZ_CP026520.1_prot_WP_042227637.1_2886 [locus_tag=PC41400_RS14670] [protein=hypothetical protein] [protein_id=WP_042227637.1] [location=3232916..3233167] [gbkey=CDS]
MDDNIIRFPVGFYQRKKNELGAKALICEVSPRWMEIVNTSHEKDFIANVMTLGADKKEKKLCELVLLKEDIYKLAELLKT
DKT
>lcl|NZ_CP026520.1_prot_WP_042227635.1_2887 [locus_tag=PC41400_RS14675] [protein=hypothetical protein] [protein_id=WP_042227635.1] [location=3233202..3233621] [gbkey=CDS]
MLKDLSFEEIPKFFEELAMKDTGRFQLSRIYGMAKSFLEQREKEESIEKLIVKDYRSVVNTTIISEDLAIVEAEVRLNKG
KETAFYPVVDNKFFNESRSTFDEALLLGFCRKYSGERCDSAIFNMLRMDLYMNRTVDEN
>lcl|NZ_CP026520.1_prot_WP_042227634.1_2888 [locus_tag=PC41400_RS14680] [protein=hypothetical protein] [protein_id=WP_042227634.1] [location=3233676..3233969] [gbkey=CDS]
MIYSVHCYFHKINSRKAGSKFEGIVFAKNKEHAEEIVRALFSKYPIEIESMSAVGREDRTLDEVYTERPELIGISPERGY
LYNEYTHKVRISKYAGK
>lcl|NZ_CP026520.1_prot_WP_042227633.1_2889 [locus_tag=PC41400_RS14685] [protein=recombinase family protein] [protein_id=WP_042227633.1] [location=3234457..3234774] [gbkey=CDS]
MSNSVMIYVRVGTQEQAEEKLGKQIVGCKQFAEDNDKRVIAVFKDVISANQRSDRFENILDEMKNQGINELLVQNWSRVS
RNTKIVYEIDQMFRDQGSFIIPIEQ
>lcl|NZ_CP026520.1_prot_WP_042227632.1_2890 [locus_tag=PC41400_RS14690] [protein=hypothetical protein] [protein_id=WP_042227632.1] [location=3234820..3235275] [gbkey=CDS]
MSTPITLDYLIKNIDQPLMNLLDIKDDFRNETPVEDLFVNPGANRETRVINALRRGGICNLENVMNVKFSYIYRLRNMGK
VSITVLLNAIVNHYHINSLIPCLKSRSDYQEEYKNIVCTIEPILLQKISTCMFQNLSLEQQRKLLKLITGQ
>lcl|NZ_CP026520.1_prot_WP_156972794.1_2891 [locus_tag=PC41400_RS29795] [protein=hypothetical protein] [protein_id=WP_156972794.1] [location=3235310..3235483] [gbkey=CDS]
MFTEIKPENRFTTEEMDEIYGLMRENDLLRRFIIALWDDRNSVMKTALEWRGNSART
>lcl|NZ_CP026520.1_prot_WP_053228655.1_2892 [locus_tag=PC41400_RS14695] [protein=hypothetical protein] [protein_id=WP_053228655.1] [location=3235519..3236223] [gbkey=CDS]
MSIDDNFKNTKGDIVHATIKSTISAIPAVGGFLSEYFGLVVTSPAEKRKENILVMLDQRLNELNNKIETFDIYSLSNNEA
FLSVVLQAINISIRTHQIEKRAALLNAVSNSALTNSIDENRQQMFLSFIDSFNEWHLKILFFLNDPKQNLQKAGHSIDLD
KGSIGLVIFQYYPQLSGELEFTKLIMSDLYNRGLISKDSSDLHTIMGGSGMVASRTSALGKQFIQFINTPDELS
>lcl|NZ_CP026520.1_prot_WP_042227630.1_2893 [locus_tag=PC41400_RS14700] [protein=DNA sulfur modification protein DndB] [protein_id=WP_042227630.1] [location=3236435..3237730] [gbkey=CDS]
MLRDRSELESNLTNAIERIKYKRKNIEEVNKNLSEYDIPSGFFNEIIKNENLLGEIDTAVLCLISMAVFKIDGSDEVRAE
NYFTEGEISEARKYTGKEVKDVNLPISINSVLQIDHENFVTTIKISEPVKWYHNKIIVYDFETQRSAKYKKGRDGVVPVP
DVNLQSVKDIAEHMLNETYLPDMITLNVYSEDFDPITYNPKSKVLTIKEGATISILDGFHRLQGAVRALSVNPDLQQDII
LSIRVYDTDTAKKYFGQINTINVVKTERLEQLKQEKVSYVAVKQLQINSDLKGKIASGSKISELAGHYTTTDILANAIDQ
VFEPKTAFEAKEVGVYLTDFFNNLLGVFDDEFNTKTSLLRQPRMFIGYIVIAKIFKENDVSFSKIKNVIETIIEEVKELD
KVVTYNKGASSRLQKLVKDHFEKIDIKSWIS
>lcl|NZ_CP026520.1_prot_WP_042227629.1_2894 [locus_tag=PC41400_RS14705] [protein=hypothetical protein] [protein_id=WP_042227629.1] [location=3237751..3238752] [gbkey=CDS]
MSNKIYEGGYFNEYQKMRYVNSLPETTSLQARRILSRAKSIEETYGQDLYNFTLDQIENLLHYLNPSTLNASRSNFYMVQ
NYIRWGIEQDLRDTNINPLDVLAENNYFKKFIDISKKSLYTKDEIDSMISVCVNAQDAFPIIAIFEGVFGKSGYDELLNL
KRQSIVSENLISLTDGEHERVINVSSDFMNLMYKALDQKTYIKKNGILKPTMRSSAEANLEQSDYVLRNVITRSQTNGKA
DKFLIQRRIAAVKSYNDDWVRVLSPYIIRNSGMLYMAKSFYDKHGEIKKDHYEEISEKFGISKVMNNGYEVYNYHRLKED
FLNVETIIKVYEQ
>lcl|NZ_CP026520.1_prot_WP_042227628.1_2895 [locus_tag=PC41400_RS14710] [protein=accessory gene regulator B family protein] [protein_id=WP_042227628.1] [location=3238835..3239437] [gbkey=CDS]
MCDLLRTTILKVDNGRSGGRLVYYLAETLTDWILNKDTLNKLKKDELERGLKKIINFGLIILLVLILSVVTNRVIETLLG
MVLLGVMRAILGGHHLPSSDLCVLYTVTLIIVIPYIGHAVDDYKLLMDSLSFFLIILFGPFRKGMSFNKNFLLYKIIGLI
IVTGNIYIYRSAIISAAIFIQAIHLIYFTKGVHDNHVDCG
>lcl|NZ_CP026520.1_prot_WP_129112490.1_2896 [locus_tag=PC41400_RS14715] [protein=LytTR family transcriptional regulator DNA-binding domain-containing protein] [protein_id=WP_129112490.1] [location=3240221..3240829] [gbkey=CDS]
MNQKLESEKAIKPIGPLVSNIKGTKKVKVINMSIIKFVREALFNKVGSLLCESDVLEIAEKVNSEYKPRTIMTVEDNALI
WIDINDILFVDKPGKETMVHTITKTYKYHDKHLDLIGMISEEEGFLFVDKRLAINIPKIKKFDSKLRNIYFTEADYTKHP
FVAGIQDRVIRDFLKTHLSKDMDVCNDRFEYSPIGIKKTTFH
>lcl|NZ_CP026520.1_prot_WP_042227624.1_2897 [locus_tag=PC41400_RS14720] [protein=hypothetical protein] [protein_id=WP_042227624.1] [location=3241233..3241442] [gbkey=CDS]
MNDYTIGLVYSDKGCDIYFGLKTKVMNVLTAANLAGSRPYAKFYRTEDKMLLQLEYYRSEKPESSIFHV
>lcl|NZ_CP026520.1_prot_WP_042227622.1_2898 [locus_tag=PC41400_RS14725] [protein=RNA ligase family protein] [protein_id=WP_042227622.1] [location=3241769..3242548] [gbkey=CDS]
MDLKKINSITKYPSILTYHKLGERGKLTEEAVEERGYLTDSDKIFVYEKVDGENSRIIFFRNQFDEIDYLIGSREELLYA
KGDRIGNPYGNIANFLKPMAERFVEDNFPSGDWALTVVYQESYGGKTKAAKNYTDSKTQSYKVFDVFSLTVNELENLLSL
SSEKIAEWREHGNQPFYNEEERGKFVQANGLEWAPLLDEVYGSEFPVTLEETYRYLRNFEQTKVGIDAIGESEGIIARSA
DRKQIRKIRFEDYERTFRK
>lcl|NZ_CP026520.1_prot_WP_042227620.1_2899 [locus_tag=PC41400_RS14730] [protein=hypothetical protein] [protein_id=WP_042227620.1] [location=3242588..3242770] [gbkey=CDS]
MKINDKVIVSGKGRNGVVYLEHPGSYQGFSKDVISKVKIKLYNGVVEEFDQDSVRLWDYK
>lcl|NZ_CP026520.1_prot_WP_042227618.1_2900 [locus_tag=PC41400_RS14735] [protein=nucleotidyltransferase domain-containing protein] [protein_id=WP_042227618.1] [location=3242802..3243113] [gbkey=CDS]
MNTTEGFNLKKFTDYLASFDFVDEVLIIGSQVTGRAHIDSDIDVCVLCDGYDFVNYNLDWTLKKKDYEHRAEIHLIPLSK
DDSRNMKFYYDNKASSVLMYKRQ
>lcl|NZ_CP026520.1_prot_WP_042227617.1_2901 [locus_tag=PC41400_RS14740] [protein=hypothetical protein] [protein_id=WP_042227617.1] [location=3243149..3243550] [gbkey=CDS]
MSEGKINFATELWKEFKDGPASTKLGIISGIFTIFGVSGVYFYSNFAIGKLLKLELWGVAALLLLSTFLITITLLIIGGM
VYVASPKRMNLKLSYWYFVAIVLFWGLGFAVLSAAFALYYELIKMLFRDLSIS
>lcl|NZ_CP026520.1_prot_WP_042227615.1_2902 [locus_tag=PC41400_RS14745] [protein=hypothetical protein] [protein_id=WP_042227615.1] [location=3243590..3244243] [gbkey=CDS]
MKNVKMRKKSVNKRSMTFWDKIINDQEEDRNEIVDRINSLSYDDAEEQVEELIFEIDDFISTYESDYHSFMREYENVLEI
SLDNLKLKRLTLITKKELGIDVVLLLKYLNNWYEKNKNDGWLFPYITPYGYWNDKTITSIDNNEYGSNGKTHLHNCINWG
KRLEDELNNFIITNKSLESDSETNRFMVDLVEECVLPIVITKNKQLNSVLTRSPKLC
>lcl|NZ_CP026520.1_prot_WP_042227613.1_2903 [locus_tag=PC41400_RS14750] [protein=hypothetical protein] [protein_id=WP_042227613.1] [location=3244317..3244601] [gbkey=CDS]
MITWKDTTSYSQRKRKDTEPRSWTAVINKADIMVHRHIHYGSDMWLLSSRYLDLDKIELKSKDINEAKNESLDKLKSILE
HNINEMEAIIEQIK
>lcl|NZ_CP026520.1_prot_WP_129112491.1_2904 [locus_tag=PC41400_RS14755] [protein=hypothetical protein] [protein_id=WP_129112491.1] [location=3244628..3244984] [gbkey=CDS]
MKVVALYDMNCDSCGCWASTDWGVSQFYNKKSCLKWAKKSGWKTVKGETHCLGCVLKPKIRKILEKKYKVTRLEFDGSNI
EFSIVVEGNKSEVYADELKVVGLELKYFSTFSVKNISS
>lcl|NZ_CP026520.1_prot_WP_042227611.1_2905 [locus_tag=PC41400_RS14760] [protein=hypothetical protein] [protein_id=WP_042227611.1] [location=3245018..3245200] [gbkey=CDS]
MSIFEAFRMMEIGHTVRDDYGILFKKKGDQLVDSIDDGKSWYPTDEFKYGYINQKWHLVE
>lcl|NZ_CP026520.1_prot_WP_053228654.1_2906 [locus_tag=PC41400_RS14765] [protein=hypothetical protein] [protein_id=WP_053228654.1] [location=3245239..3245703] [gbkey=CDS]
MDMNKDVKQLLYKMLNEVNIYPTDEQIAIVNRGRPHKCTFKQGKMYVYTFSFNGDYLKIGKAGSNSKARFYSQHYNPESS
QSNLAKSIILDPAMEFYSLSSSTVGDWIKNNVDRIDIEIDAKLGVFTLNLIESILHCLYLPRYEGFKTQRADKM
>lcl|NZ_CP026520.1_prot_WP_042227610.1_2907 [locus_tag=PC41400_RS14775] [protein=hypothetical protein] [protein_id=WP_042227610.1] [location=3245991..3247133] [gbkey=CDS]
MTDKSIVKTNKDHGGKEFTSSCHFVGKVKPVRNQENGTDNWVDVPYYREETDDNGKIKKILEFIIETAESNDLKVKLSGR
EMPFVYPYSSSKKKSVKLSWEDRFDKAKYPDDTYHYIDSEWDKIDKFKELIYTDAWIEVKGKHNPYEFTDERDNEIKGVS
RIPSYINPIIDGKVKIGDELRAIQVNKVDIPYICDLKSPDFVEVNNFKMQIGIRSTYADKEKGMTNVNGVFLTYGKTSSE
PKNVHLIVYQQEASEGKKPLADAFESLDEFDFIEVTGQDNNRATFTWLPIKEKLDDNDPFADVDESNKQVSYEKVTNGAK
KGLEITGYVAGSLIREYLTEAELVKSVSTTQEDPLKTENKKDPFDDNAHPIDFSDDDLPF
>lcl|NZ_CP026520.1_prot_WP_042227608.1_2908 [locus_tag=PC41400_RS14780] [protein=AAA family ATPase] [protein_id=WP_042227608.1] [location=3247193..3248419] [gbkey=CDS]
MSFRNLVTANIPKVDLQSYILLTAGNYKSGKTRWWKEIIEYFFPNEPEAGLLVAWEPGYKTWKLKSKIDLVEYKKEEEKI
KNGKIKIKTDIEMDSDRWIYFKQEIVTGLVQESKVSRISKVVGFDTVDRMIDCASAYIIVNANNKYPGNNFTSIQELSES
KIYKDNVWNNLYDELKRPIDTLRAAGYGLIFLAWTKEKTTELVNGVKYNSIELMMNTTCRKVFQSQADLICCLHNEVTAL
DKEGNELNKNLENKNGKEIATKFHESQTYMYFRESSYIGIAGGRFKVLPEKELYGIDTYVRVFKEAIEGQLDEGDSFEDI
RKKEIEVLEEKAKRFSEDMIDSLKDEDEKTAEDYHNELKNALKDIANEVKSDIVAPKMKELLGVVNFLKSDDTEKLKEAL
DFVNALNQ
>lcl|NZ_CP026520.1_prot_WP_042227606.1_2909 [locus_tag=PC41400_RS14785] [protein=3D domain-containing protein] [protein_id=WP_042227606.1] [location=3248522..3249109] [gbkey=CDS]
MSENKTLSFIFMAVLIFLTGCEANENVKDPKETTQPTIEMPLKPIRNLVHLEIQSTPAPSLEPIPAATEFDVKSIEIEDK
NTVTKEVKHKDKYMNFEVTAYTNNRESTGKSPGDKYYGVTASGQTTKEGVTIACPKSMPFGTQIYIPYFNNTYTCTDRGG
AIKEGRLDVFMSSESKAVKFGRRKLEVQIIEKKGE
>lcl|NZ_CP026520.1_prot_WP_042227604.1_2910 [locus_tag=PC41400_RS14790] [protein=hypothetical protein] [protein_id=WP_042227604.1] [location=3249112..3249384] [gbkey=CDS]
MSYVANERETVIQCDDEKMKWNLYTLQPRVISKLKKAGIEPVRINEDGAHYYEEIDFGRVSFRNESTRTMSDEQRQAAGL
RLKKAREKKN
>lcl|NZ_CP026520.1_prot_WP_042227603.1_2911 [locus_tag=PC41400_RS14795] [protein=hypothetical protein] [protein_id=WP_042227603.1] [location=3249381..3249998] [gbkey=CDS]
MNNNKMKIISIGGLDKSGKHTQSKLLTERLRSEGFKVMSSEFHRYETPTGELIMKWLRKEWEVSAETIELIMAADKQAQQ
DWFTELENEKYEFLILDRYSLCQVAYGLANGRDGRWIMELQKYMRKPDLDIIIDIPAEESMKRKGKHGENDRYESDLGLL
TRVRSNYKNVSTNYSAPAKAIVDGLRSVEEIYKDIAKIVNKEFRQ
>lcl|NZ_CP026520.1_prot_WP_053228653.1_2912 [locus_tag=PC41400_RS14800] [protein=hypothetical protein] [protein_id=WP_053228653.1] [location=3250843..3251214] [gbkey=CDS]
MSEIPKLYRILTQTEAAMITADTTTDMTELEAKALAQKDIIAPNGNYQPFVLIPICTEFQWRYGQINFCPRCGKNICESV
GGDYLEGVTDYLEFDCPECEATMFVHIISTPEEANSIDVEDRN
>lcl|NZ_CP026520.1_prot_WP_042227602.1_2913 [locus_tag=PC41400_RS14805] [protein=DUF2493 domain-containing protein] [protein_id=WP_042227602.1] [location=3251228..3251581] [gbkey=CDS]
MKVIVAGSRTFSDYEMLKRKLDLLLALQPRNEIVIVSGAAKGADKLGERYATEKGYRISSHPADWEKCGNNAEYIRNEEM
AKKSNALVAFWDGTSKGTKHMIDLAHKYELQVRVVRH
>lcl|NZ_CP026520.1_prot_WP_042227601.1_2914 [locus_tag=PC41400_RS14810] [protein=hypothetical protein] [protein_id=WP_042227601.1] [location=3251612..3251899] [gbkey=CDS]
MPIKTDNNIKYLQFGYGDIFVAELKDDGSKLTYGAAFIQSKPREIGEEWKEVVGTSLDNHDAHVIMLFDRIASLDVVIET
LKEIKEEMIKQGFDY
>lcl|NZ_CP026520.1_prot_WP_156972793.1_2915 [locus_tag=PC41400_RS29800] [protein=hypothetical protein] [protein_id=WP_156972793.1] [location=3251934..3252110] [gbkey=CDS]
MDYKIDQNFYDIIDGREVEITDIHLDKDGNIEGFSLKKGINQWFSTIRSFQEYYRACQ
>lcl|NZ_CP026520.1_prot_WP_042227600.1_2916 [locus_tag=PC41400_RS14815] [protein=hypothetical protein] [protein_id=WP_042227600.1] [location=3252296..3252871] [gbkey=CDS]
MNLDDMRKEYELAKQDRQNAKSATARNGIRDALYILTKKIDAEEIRVNSNLKKVEIGGVTYNLPTSFSNRGLEKKILYSV
GEIMYFIDENNTYETDGSYCWHHYAWVPQRKDKYVRLLVRTLGGDHFGDRFFTETSYYKHPADPYPYLTKDIYVDNRTYS
PHVKFIISKIGLEIDKTSREANKLVKILKES
>lcl|NZ_CP026520.1_prot_WP_042227598.1_2917 [locus_tag=PC41400_RS14820] [protein=hypothetical protein] [protein_id=WP_042227598.1] [location=3252871..3253137] [gbkey=CDS]
MKKNFIVSVLSLLSVLIAACGQTETEIKRATVDSQEFSVVSVDQADTASLITVKHQKTGCYFALSTSDRNGIGSLAQIYD
QSGKPFCE
>lcl|NZ_CP026520.1_prot_WP_042227596.1_2918 [locus_tag=PC41400_RS14825] [protein=hypothetical protein] [protein_id=WP_042227596.1] [location=3253409..3254572] [gbkey=CDS]
MKYTFEKNKLYDYLGKHVVNAFKRHEVYVAGGAITSLFSNRKVNDIDVYFRHEASLIEFVEEAWEGSDWVNILTNKSIMV
RMGRDKNVQLIHFKYFPEAKDIFDTFDFTSCMGAFDFKTEEFVLHKHFLKDNAQRMLRFNKDTAFPIVSLLRVHKYTEKG
YTISKPEFIRIALKCMDLQIKSVDELKEQLGGMYGINYDKLIQFEEGEEFSLDNVIDKIADLSLHEDYFKKPEEVKFECV
EDIIKEISKEPVQITYINERDYRLTRKGTLKFISDIPKKYTEFDGKTYIENKRFYKFVKYNKDGNYSSHYDSNFIYKIGE
FAIPKNDYLYFNEQKEINESNYRYQGALIEVIIPYEHFDHKDDAKVHAKKCFVVREVPREEYMSWID
>lcl|NZ_CP026520.1_prot_WP_042227594.1_2919 [locus_tag=PC41400_RS14830] [protein=hypothetical protein] [protein_id=WP_042227594.1] [location=3254603..3254950] [gbkey=CDS]
MKPMTEKQRKAIWGIENILDISYKGNDTAQEAFFFLSEHLPTAQEQVNRMIQNSAIYQTGNDNRFSPNKSIMDQMSFYRF
DNSNNLSDPEDDDSFAETPTLESLDADLAFLYRLI
>lcl|NZ_CP026520.1_prot_WP_241688657.1_2920 [locus_tag=PC41400_RS14840] [protein=site-specific DNA-methyltransferase] [protein_id=WP_241688657.1] [location=3255191..3255931] [gbkey=CDS]
MKLLELNKIYNVDCLGENGLSLINNKSIDLILCDLPYGTTRNKWDSIIPLDVLWRHYKRIIKDNGAIVLTAQTPFDKVLG
NSNLDWLRYEWIWEKTTATGHLNAKKMPMKAHENVLVFYKNLPTYNPQKTIGHKPVNSYTKHQDDGSNYGKTTIGICGGG
STERYPRSVLKFSTDKQKLAIHPTQKPVDLFRYFIRTYTNKGDIVLDNASGSGTTAVACLEEERNYICFEKEREYWEKSI
ERVSKK
>lcl|NZ_CP026520.1_prot_WP_042227590.1_2921 [locus_tag=PC41400_RS14845] [protein=hypothetical protein] [protein_id=WP_042227590.1] [location=3255972..3256310] [gbkey=CDS]
MGTDRSDYIIIGCDVKQLVDKMDEEGKDNFYDGYYRWNRRNGEIVHLPDNNSGDYDYFGVIVEHDPDGYEGLSPFIYEEN
RYAEHKEKVTKAIKEVFDVELEPKLIVLTHWQ
>lcl|NZ_CP026520.1_prot_WP_241688658.1_2922 [locus_tag=PC41400_RS30315] [protein=hypothetical protein] [protein_id=WP_241688658.1] [location=3256628..3256870] [gbkey=CDS]
MGRQYVFEDVLTFTGYGRGRSSAVMHFSGSTGAEYNMFLTDTEDLIKAKDIIDGRVKAHWTFCKRGSNYGIRLADIELGK
>lcl|NZ_CP026520.1_prot_WP_129112492.1_2923 [locus_tag=PC41400_RS14855] [protein=hypothetical protein] [protein_id=WP_129112492.1] [location=3256907..3257104] [gbkey=CDS]
MNLWQRLEDRGREELKLGQSWYLYAVDLDKSGFGWSEVKFARKNTQGMFLTSDSVWGNIKYSDLR
>lcl|NZ_CP026520.1_prot_WP_053228652.1_2924 [locus_tag=PC41400_RS14860] [protein=hypothetical protein] [protein_id=WP_053228652.1] [location=complement(3257162..3258334)] [gbkey=CDS]
MKTDGTEVAQLFEMNDIDSRLRHLNHDQINKLISRYYDGEKIETLLKEYQISARASELYKMFPATIENIKCFYCNVHLVT
MMPSRTSNAKNYAFCPHCDHKETPLCRCTNCKKLEQEKILSENQYKKIKIAETYDIANYKSNQINDLNLRDVLYLACLLR
SRLDENFSIIQPLENYIESITPTSDKTKELVLSLANKKIIVPHSSSPINSFKEDEDFPNTYYIYRVNYQVNVEEELSVLL
NPDSTIFLNDPHFVYKMWREIALEETKQYLLYSMKKVKFDFSIGDKTISVFNDLLDHFSTSQIHALIYRGVSNATRYYQE
GQVPKQQAANSVITNCQRLGEKAIAENWDVKGFNRNYDLPQSVLSELFFNRILKIGSRGFDEVPDILIDS
>lcl|NZ_CP026520.1_prot_WP_042227588.1_2925 [locus_tag=PC41400_RS14865] [protein=hypothetical protein] [protein_id=WP_042227588.1] [location=complement(3258427..3259101)] [gbkey=CDS]
MSIIKRGFIYALSDIHFHTLSNFSSDIEHSIKTSSAEYIAHLEAISENYKDDNLEMFWEYNIDQVNDLRYEYPTLLRSSV
LISCYSNLEKTMVNFHEDLKLKTGLSLNYNNRRFSRRSTIEKVLESLQESIPLPNSSGSIEWTNILFYKSLRNKVVHDLG
KINKRKDRALLNQIEQSSFVSVNEINTITFNEHFSEKVITDCRNFIDLLYTDIKLYTDSLRPSS
>lcl|NZ_CP026520.1_prot_WP_042227586.1_2926 [locus_tag=PC41400_RS14870] [protein=hypothetical protein] [protein_id=WP_042227586.1] [location=3259509..3259991] [gbkey=CDS]
MEQIKKEILKLNLVLENTDCIEIENKHIGRISILDIKTSIVATRNSTCKFNECDHFALASFRDGDEQYKLPNDFMSTSSK
YTRLKGNDITSIHIIYNDYTEEELYVPWGDSEYKNDYQHTYINEHGDLFIVINKNKNIEDEFPYDLEDTACLEYMLFWDC
>lcl|NZ_CP026520.1_prot_WP_164977127.1_2927 [locus_tag=PC41400_RS14875] [protein=hypothetical protein] [protein_id=WP_164977127.1] [location=3260008..3260322] [gbkey=CDS]
MMEKLKCKDPECIELHETEWEDIELILNVEEAQTTQYKIGEDGILKVIDSDCTAQELWLICPCCDTRYEFKQEYVEVEKC
AKLTFDIDDNNNFKINRIKKAWED
>lcl|NZ_CP026520.1_prot_WP_053228651.1_2928 [locus_tag=PC41400_RS14880] [protein=Rha family transcriptional regulator] [protein_id=WP_053228651.1] [location=3260548..3261420] [gbkey=CDS]
MDFNIINHNGNLYIESRQVSEMIEKPHNDLMKSIRIYAQYLTQGNISLSNFFIESTYQDTTGRTLPCYFLTRKGCDMVAN
KMTGEKGTLFTATYVTKFEEMEKKLSQLSPLEVMQLAINQLVDHEKEIKNHGKEIDSLKENGLMLTQDINSLREDFEKYT
IRIGDKSGYIVARELKLFSSSDNPHFNFVDAVAKKLNIYSGILGDKNEYVNVIMDNTHNGNSGIAVYYTEVAVQLIKEYL
AENFKLEVILYQRSTGGFKKGEPKEFIFKLNNKNWQFNKKTYEYFYNLTQ
>lcl|NZ_CP026520.1_prot_WP_042227582.1_2929 [locus_tag=PC41400_RS14885] [protein=hypothetical protein] [protein_id=WP_042227582.1] [location=3261665..3261916] [gbkey=CDS]
MKLVNADNTYAIIRGLRLNNRFTIGFGWSRNKGDWWWKDSPQNVRLFDIRKTMGNEITKYGIYIARLSMVLMKKSGGGRS
EYE
>lcl|NZ_CP026520.1_prot_WP_042227581.1_2930 [locus_tag=PC41400_RS14890] [protein=hypothetical protein] [protein_id=WP_042227581.1] [location=3261909..3262562] [gbkey=CDS]
MNKTKLKYVEFRLDLEKKILAELMPSHLRLDFLKHSYIAGGCIYSIYQSQEPKDYDFFLTDESFASQLRKHFLEKESGYH
GKEVSGGMYLDYPLTITENAISIGRYQIITRWIGQPKEVVKEFDFRHLQFYYVGNGIETVTQYDFLDSRDLKYNEDRARD
ICGSVFRSSKFVARGMKISQKEMAKMLLRLKNVGFSEKEVETLIACKEKPDADHFGS
>lcl|NZ_CP026520.1_prot_WP_156972791.1_2931 [locus_tag=PC41400_RS29805] [protein=hypothetical protein] [protein_id=WP_156972791.1] [location=3262578..3262748] [gbkey=CDS]
MRRAELKGYIIFNEDKLNHGNNIIGQIDHELFNVDGILEWELEEISNEEVEHIDED
>lcl|NZ_CP026520.1_prot_WP_042227579.1_2932 [locus_tag=PC41400_RS14895] [protein=hypothetical protein] [protein_id=WP_042227579.1] [location=3262832..3263215] [gbkey=CDS]
MVAASCNHEDLSCIKEAALNPILLDVLERDIKLMMDSEIKMKQIYARKLKHIQNLISEDFVDTKKDLKNRGLKIYEYKRD
SKGVYAKFLCRGYHHEFSMLGVLIKSEVELRLAAYLAMDLKDDKTEI
>lcl|NZ_CP026520.1_prot_WP_042227577.1_2933 [locus_tag=PC41400_RS14900] [protein=hypothetical protein] [protein_id=WP_042227577.1] [location=3263279..3263812] [gbkey=CDS]
MGMVKCQYCLVKGDKTLMEKGIHKKGYIHKNCIVDFRDNEKKKAVEQEQRRELYSYLLCLHNAIEIPARNLLRLKEIKEN
KNIEYKLILEAYKISEEKIKWFISNVLDYRNDSEGINACITLMLKHGINPAFQERKNKAKQEIHQKEMYEVKTRADMSLG
AAARSKVKDEMDISHLL
>lcl|NZ_CP026520.1_prot_WP_042227575.1_2934 [locus_tag=PC41400_RS14905] [protein=replication protein] [protein_id=WP_042227575.1] [location=3263825..3265327] [gbkey=CDS]
MSTKHITEFIAPSEVHESLFVGYLWKTPTLYSKYKIHKIDKTTFTRPIWYFYYYVGKEMFESGLRSFEDTSVYSFLSSKP
KEKGKADWLEVYNEFGAYSTIAELVEECNPEKGNEDYHLAEIQKYESLRKLQDEGFINVLNLELIHKLTSMSLSQLQTFY
QTKFKSSFAQINAGEVVEYYLGDNLNETIEELKKGIQSGIPFFDSPRLNKKINGLKLGNLNYLVLPSGVGKSSILTEKAV
LGIYESDEKAIVFANEEGIRRWRSRLLATVASRILKKPLARDIIERGTFTDEGEAILNEAREWIEKHRKENILFINLKKY
RVQDVIGRIELYRARGYKHILFDTFKPDLSQQIERWLAFSNSAQDLYDCIKEEAYNCHCLATVQLKIGREYRFIDLDCIG
KSLEIVEVAAVVMAGRLMFDDEYKEDGHKNRLYPYNWKKDDFSGEWIPIPYKLDPKKKYLILFLPKNREGSEDEQIVFEV
NYDFNIWREVAYVKVPNNGR
>lcl|NZ_CP026520.1_prot_WP_042227573.1_2935 [locus_tag=PC41400_RS14910] [protein=DNA primase] [protein_id=WP_042227573.1] [location=3265348..3266421] [gbkey=CDS]
MINDLMLIKKRIITENRIEEILSKAGCENIKEKSNRYESMLPDKFESDNSRGLQVYLNESLTCKIRNRTFLGSDIFDLIS
YIVFDKIEALDIHKCLPKSKRWICEQLGYHEYLTGEQVVVFNDPLKWLKDIKKNRSKSILEVKENKVLSDDSLDRFVMFP
HAALIEEGIEYGIQQEFQVGFDLKSERIIFPIHNSYGEIVSIKGRTTDPDYKIKDIPKYLYLHNFNNMWELYNWHRALWY
IIESKEIIIYEAEKTCWLSTQFGVRNCVALGGSEVTDYQARMIKSLGIDIKIVLAFDRDKKPEEIKLQAQKFGKSRSIFV
MWDGKSVFTVEAKHSPTDLGFQSFMDLYKDHYNYRIS
>lcl|NZ_CP026520.1_prot_WP_042227571.1_2936 [locus_tag=PC41400_RS14915] [protein=DUF4176 domain-containing protein] [protein_id=WP_042227571.1] [location=3266565..3266927] [gbkey=CDS]
MNYLSEQERIEDEIYYAQWQLEHPNSKYKHEKINNKTEDQMEECLLNIGSVVKVAPDNDEQQEWIIIGRRIVSPKSLIAW
DYVSVSTKGFQWTVSQDKSFSPYFFFFNHPDIEEVVHTTK
>lcl|NZ_CP026520.1_prot_WP_042227567.1_2937 [locus_tag=PC41400_RS14920] [protein=hypothetical protein] [protein_id=WP_042227567.1] [location=3267181..3267549] [gbkey=CDS]
MPRIEIEVRQVGSMSTWKEKYDFHEGDPQAWAQAMIDRFNSKLRPGENPRELVDVEVLPEESIVEHLWEKQNTITIIRGA
HIYDKMRCERCGVTGKRHGLSSGIKRDSEYRAKKYEKCTGHV
>lcl|NZ_CP026520.1_prot_WP_042227565.1_2938 [locus_tag=PC41400_RS14925] [protein=hypothetical protein] [protein_id=WP_042227565.1] [location=3267579..3267869] [gbkey=CDS]
MSRTTFNSSREAYAYILEEQEKAKELLERTRLDISFLWMNITKGNFEDDSKQEHALKELGDLAKSSYSLADSAARLSNLA
KALSAYFADEINGNVD
>lcl|NZ_CP026520.1_prot_WP_042227561.1_2939 [locus_tag=PC41400_RS14930] [protein=hypothetical protein] [protein_id=WP_042227561.1] [location=3268138..3268422] [gbkey=CDS]
MAIEAHKCNQPECKGFVLIENADFDLKDIPLDEKYGCYAFDRPHCSECGKEFLVVPHYIVIEVNDKDFSEWEQIESTCMT
QFERRQRELKMLSS
>lcl|NZ_CP026520.1_prot_WP_042227560.1_2940 [locus_tag=PC41400_RS14940] [protein=hypothetical protein] [protein_id=WP_042227560.1] [location=complement(3269119..3269928)] [gbkey=CDS]
MKKLALSLTICSVVLSSIVPVAAASSISETTLNDYLSKARYPKNVINILDKEQKETIYNNKAEFAFVKSGSELLTDGQEK
SLITPQELANFSAQLVGSQISSPSGYAKIILDFNFDWNYMPVFTSKDKFGIAWSDDFTALPETAKYNYKAVNNASGLYAE
LGNTSDYSDYNIKKGIGWEINLTAGFTRNGTFYPVGRHKGWGQVIIQRPKDKPGTPMTTAAVGNYFHKEIAGNGSLSFGL
FDGKPQISISPSISFDKSNDYADVVNWTQ
>lcl|NZ_CP026520.1_prot_WP_042227558.1_2941 [locus_tag=PC41400_RS14945] [protein=hypothetical protein] [protein_id=WP_042227558.1] [location=3270124..3270360] [gbkey=CDS]
MIEISQLVGVNSFDEATHISVDKDLSNITKDKIYELYVGKPIGISCGEDEHYIIDDNGERMHGVWAFIKTKYYKQNNT
>lcl|NZ_CP026520.1_prot_WP_042227555.1_2942 [locus_tag=PC41400_RS14950] [protein=hypothetical protein] [protein_id=WP_042227555.1] [location=3270510..3270929] [gbkey=CDS]
MGMFTQVRGWLNIDSIGDFNKENYRALRNKLNEVRKDYLDFAEGRACISEDTNIHLGANNSIFLFIGTELKNYDDDAEEW
IKYLIARFPNAEGRIDFQYEIEDFRDQDSKSKYWLIRGGEIIEEGYCKTWCTGYGNTID
>lcl|NZ_CP026520.1_prot_WP_042227551.1_2943 [locus_tag=PC41400_RS14955] [protein=hypothetical protein] [protein_id=WP_042227551.1] [location=3271205..3271495] [gbkey=CDS]
MIKWTIVLALSALLLAAGCKAKPVKIIKDLEYVEQIKTSRGDILQFIDIEGYKINYFVDPSQSYNLEKGKSYDVTVTIRS
AIQEKDFVNDAILSKQ
>lcl|NZ_CP026520.1_prot_WP_129112494.1_2944 [locus_tag=PC41400_RS14960] [protein=hypothetical protein] [protein_id=WP_129112494.1] [location=3271528..3271791] [gbkey=CDS]
MKKGQLVEISYKHPGGWEYGVVSKIYDDRCLILKEYGGRLMCEFTQPNSEKNPNEQWLSDDVKVAELFVDGKPVMAKKNL
LGGYKFL
>lcl|NZ_CP026520.1_prot_WP_042227547.1_2945 [locus_tag=PC41400_RS14965] [protein=hypothetical protein] [protein_id=WP_042227547.1] [location=3271845..3272345] [gbkey=CDS]
MGFCYKAKSGSEFYLDARKSMTQRGEWKKVINEVNKLLGESVKSIWPSTNILCLDVRELSKDENKKLFTNEGRLRKNDKK
AKDYNSEYIKILNRFGLSNYEDIKLVEFKHGICSLGGESLERYISLDKEIYYKADFNLEKRSQGNFDLITEIEYQEKYLE
DLKKSG
>lcl|NZ_CP026520.1_prot_WP_129112495.1_2946 [locus_tag=PC41400_RS14970] [protein=hypothetical protein] [protein_id=WP_129112495.1] [location=3272356..3272562] [gbkey=CDS]
MKKAVIISCDSPLNWWRRHRGRICEIVEEENEYGYIKIRLLVNDNTSFEAKQREIIGGIPVKHIRTVS
>lcl|NZ_CP026520.1_prot_WP_042227546.1_2947 [locus_tag=PC41400_RS14975] [protein=hypothetical protein] [protein_id=WP_042227546.1] [location=3272593..3272937] [gbkey=CDS]
MLEEINTYDWKEAFGYANSVFTVHFAKPVSTRPFSREDVVEIIAMDDGENDTSNWIGIFKLKDGRYAIIDAGCDYTGWDC
QAWGSAEVTGSLEEAIRFGLDNSQRNRLNLRINE
>lcl|NZ_CP026520.1_prot_WP_164977128.1_2948 [locus_tag=PC41400_RS29810] [protein=hypothetical protein] [protein_id=WP_164977128.1] [location=3272965..3273129] [gbkey=CDS]
MLEISKVADTLANQAAIFNCTFKKAWKYYMHPAITENFSYEDIVFYINVKMKLG
>lcl|NZ_CP026520.1_prot_WP_042227545.1_2949 [locus_tag=PC41400_RS14980] [protein=hypothetical protein] [protein_id=WP_042227545.1] [location=3273172..3273648] [gbkey=CDS]
MPRPRVYKLEHLAVVEAGQDPKIYFKTTWSDEDLTMLIKVWMYKIADYVPEYEFYEEEMMELLETFGHKQISAYSPFRTY
FSADMYEIWEAANIRPTVEDESNPKFINDNAMKVFERVLKRNDDEFLEYGLKNDNYKKYNEHARRDFDKVAARNGITV
>lcl|NZ_CP026520.1_prot_WP_042227544.1_2950 [locus_tag=PC41400_RS14985] [protein=hypothetical protein] [protein_id=WP_042227544.1] [location=3273645..3273917] [gbkey=CDS]
MNQDIHSYFGLSYANYLVLPRTVLQSMPMEWQIKFVELLEEVDDTNWRTELMPKNWMDYLVKVRDKNGRFISDPLDNYDR
GRRRVFDTDQ
>lcl|NZ_CP026520.1_prot_WP_241688659.1_2951 [locus_tag=PC41400_RS14990] [protein=hypothetical protein] [protein_id=WP_241688659.1] [location=3274391..3274630] [gbkey=CDS]
MKKIARKYDGVKSESFRIDIYNAKEVSDKGSQVIKTIDNLTDLPTIKRIENLAKENGLDTFEITKITERTTRYIVDREK
>lcl|NZ_CP026520.1_prot_WP_042227542.1_2952 [locus_tag=PC41400_RS14995] [protein=ATP-dependent Clp protease proteolytic subunit] [protein_id=WP_042227542.1] [location=3274627..3275154] [gbkey=CDS]
MIKTNDRTLYLYDKICPEAAGKLIEQIININERDDEEKIVDRQPIKLIINSYGGQIYDGMAIVATMLSSKTPIHSYVYGY
AMSMALLAAAAAHKRYAHKLATFMYHDSLTNLEGKLGHVHDTLGELRRVKNMYDDVLLTKTKIPMSKLDEVRKMKDDWYI
TAEQGLELGIIDELI
>lcl|NZ_CP026520.1_prot_WP_042227541.1_2953 [locus_tag=PC41400_RS15000] [protein=PD-(D/E)XK nuclease family protein] [protein_id=WP_042227541.1] [location=3275235..3276032] [gbkey=CDS]
MSEIIWSWSRLNTYHSALEGDGCFHLFNQQYNIGNRGENNYFAEYGTLVHEMTERLHNGSLIAWDIEDELKRGLKNFHYK
VPFQKMRASYENSLFKFFDEFEDVFKDYTITQAEELKNFEVDKIKLKGFPDMFAQHREYGKVIGDYKTSRVYDTKKMKHN
IMQLYLYSIPYYNEFGVYPDNLVYIFPREKENRERVFKFEIEELERTKQWVRDTVCKVLNHKDTWTPRCKTVDGKKDFFA
SQLCNLRNQCAFRYSCVHRNEFTPY
>lcl|NZ_CP026520.1_prot_WP_042227540.1_2954 [locus_tag=PC41400_RS15005] [protein=hypothetical protein] [protein_id=WP_042227540.1] [location=3276055..3276294] [gbkey=CDS]
MKFYAIVNNEEGKSVILESTENTRQLAEKDVLAQCKDRGYKFKHLHLLQGSQKQGGVNTKFFKDRKRTGKNGKRYASFM
>lcl|NZ_CP026520.1_prot_WP_042227537.1_2955 [gene=nrdI] [locus_tag=PC41400_RS15010] [protein=class Ib ribonucleoside-diphosphate reductase assembly flavoprotein NrdI] [protein_id=WP_042227537.1] [location=3276388..3276741] [gbkey=CDS]
MIIIYDSKTGNVERFVKKLNLECYRIVPQLKVSTSFILVTYTTGFGEVSRSTKDFLEANHVHLKGVASSGNRNWGVKFAK
AADIIADTYKVPAILKFELSGTEKDVAKFMLEVMNFV
>lcl|NZ_CP026520.1_prot_WP_042228802.1_2956 [gene=nrdE] [locus_tag=PC41400_RS15015] [protein=class 1b ribonucleoside-diphosphate reductase subunit alpha] [protein_id=WP_042228802.1] [location=3276776..3278833] [gbkey=CDS]
MVQKDGKYQFEKDKEAVKSYFVDHINQNTVFFHNLKEKLDYLIENDYYEHEIINQYELEDIKKLYKHVYGMKFRFPSFMS
AFKFYNNYALRTEDNKKYLERYEDRICIVALTLAEGDTDKAFWFADKMIKREYQPATPTFLSAGKKRRGELVSCFLLMTD
DSMNSISHSINNALQLSKRGGGVSHNLTDLRALGDPIKGIEDKCSGVIPVMKLLEDSYSYSNQLGVRDGAGAAWLNIFHL
DILDFLSTKKINADEKSRIKTLSLGVIVPDKFFELVEKDYDMYLFSPYGIYKEYGKRMSEINISEKYDELVENPNIRKKK
ISARDMLTMIAQIQVESGYPYLFFEDNANSVHPLSEEGKVKFSNICTEIIQLSETSTINDYHEQDEIKRDISCNLGSLNI
VNVMENKDLAGAVSAGIRMLDVVSRMTNIKSVPSVDKANRQMRSVGLGAMNLHGFLAKNKISYESKEAIEFANVFFAAVN
YYSLKESMLIAKETGFNFKGFRNSSYNGGNYFDQYLSNDFLPKSDKVRLLFQDIKLPTREDWSMLKGDVMEYGIANSYRL
AIAPTGSISYVQSSTASIAPITEKIEHRTYGDSDTYYPMPYLDESSFFYYKEAYDIDMFNMIDLYAAIQNHVDQGISCTL
YLKDTMTTRDISLYYIYAWKKGLKTLYYSRMKMTQIDDSCVSCVV
>lcl|NZ_CP026520.1_prot_WP_042227535.1_2957 [gene=nrdF] [locus_tag=PC41400_RS15020] [protein=class 1b ribonucleoside-diphosphate reductase subunit beta] [protein_id=WP_042227535.1] [location=3278859..3279836] [gbkey=CDS]
MIKPVNWNEQTSDYANIFWEQNTKQFWLDTEIPVSKDKKVWESLTNTEQEVYMKVLAGLTLLDTEQGGVGMPKVLQHVDD
LQEKAVLAFMAAMEQVHAKSYSTIFTTLATKKQIDDAFKWAEENKWLQAKAMKIDSIYEDIKDDYTLAKALIASVFLESF
LFYSGFYYPLYLAGQGKLKNSGEIIFLITRDEAIHGVFVGLLFQKIKSKLTKTDKKRIDESIHVLLNELYEIECRYSEDI
YDDIGIASEVKKFVRHNANKALMNLGYEPYFPAEEIDQIVLNGLSSGGTFDFFSLKGGSYVKAKVEPIRNGDFEYLNELM
KGYYK
>lcl|NZ_CP026520.1_prot_WP_241688660.1_2958 [locus_tag=PC41400_RS15025] [protein=HNH endonuclease signature motif containing protein] [protein_id=WP_241688660.1] [location=3279863..3280573] [gbkey=CDS]
MLKDGEILKKVDKCPWMISNKGRIKSLKTNAEKIQYMGTSGYYFVSYNSKTYLTHRLVAQYFIDPIPKGMVVNHKDGNKT
NNHVDNLEIVTYKENTRHAFDHGLMVPLCGEENSMSKLTNSQARNLINDIIKGLANGELAIKYNLHDRYISLVRHKKRWK
KLWSEFGSINAEMSNGNDRNKSLSPQQFVQVVKKIKNGVSNASIEREFNLSSGTGSRIRHKKIYCNWWKDYFGESI
>lcl|NZ_CP026520.1_prot_WP_053228650.1_2959 [locus_tag=PC41400_RS15030] [protein=hypothetical protein] [protein_id=WP_053228650.1] [location=3280591..3281094] [gbkey=CDS]
MNINENMYYQMVREFHKAFNHPVAEQPTIMDVETALNRMSWDAEESLELLHASSETIEEFQFLYLKLLANMKNTYEQLLK
TEFPEDRLSAQVDALCDKEYFNQGSFVMMGVKPDRVVEAVHNSNMSKLWNGVPKFREEDGKIIKSDTFVSPEPMIKKEIE
RQQVSSK
>lcl|NZ_CP026520.1_prot_WP_042227532.1_2960 [locus_tag=PC41400_RS15035] [protein=thymidylate synthase] [protein_id=WP_042227532.1] [location=3281161..3281973] [gbkey=CDS]
MNNVDKVYLNLGRDILVNGIEKNDRTGTGTKSIFARQLRFDLNEGFPLLTTKRVHFRSILGELLWFLSGSTNKFELSRNY
GVTIWDEWGDDETGELGRVYGSQWRTWQAPNGRTIDQIANVINQIKGNPDSRRHLVSAWNVGELDSASLPPCHYAFQFYV
ADGKLSCMFQMRSTDYFLGLPFNIASYALLTMMIAQVCSLQLGELIYSGGDIHIYSNHVDQVNKQLAREPKSLPKVYLNS
NIKKIDDFSMLDFILDDYNPHPGIKAPVAI
>lcl|NZ_CP026520.1_prot_WP_156972789.1_2961 [locus_tag=PC41400_RS29815] [protein=hypothetical protein] [protein_id=WP_156972789.1] [location=complement(3282296..3282457)] [gbkey=CDS]
MKKIKQLALWILINKEGIADITLELLKNPAIDEFIMAALKFFILKKSADNRKN
>lcl|NZ_CP026520.1_prot_WP_042227528.1_2962 [locus_tag=PC41400_RS15045] [protein=DNA polymerase IV] [protein_id=WP_042227528.1] [location=3282621..3283865] [gbkey=CDS]
MTSKTKSIIMLFDMQSFYASVEKAKDPTLTNKPIVVAGDPSRRSGVVLAACPLAKQYGVKNAETLGDALRKCPGLVVIRP
HMETYIAVSMQISNILQQYSDLVEPYSIDEIFIDCTNTMHLFANTPEEMAKEIQRKIRIETGVHARAGIGENKTLSKLCC
DMIAKKNNDGIFILRKNELYKYIWDMPIRDMWGIGSRMEKHLLRMGIKTIGDLANTPLPRLTKRWGVNGQLIWMTANGMD
DSPVTPSTHMSQKVIGNGMTLPRDYCEPWEIEVVLLDITNEVTKRARNKNLMGSVVSVGCMGADWDHPTGFHRQMKMPDP
TNVSGDIYSVVKRIFYTHWDRQPVRKLSVSLSDLSDANTYQLSLFDDCEKKRTLDSVMDSIKDKFGELAILRASSVTSAG
QAKDRAAKIGGHYR
>lcl|NZ_CP026520.1_prot_WP_042227527.1_2963 [locus_tag=PC41400_RS15050] [protein=YolD-like family protein] [protein_id=WP_042227527.1] [location=3283862..3284206] [gbkey=CDS]
MSIKLSGNGRWESSRMMLPQHKEAINQQIKEIGLKGKPELHEDELQIVMENISISFKRKTEICVTIYGEYEFQDLIGVVT
SINPYRQNFKIELVDGFEFIDFAEVIKADLREES
>lcl|NZ_CP026520.1_prot_WP_042227525.1_2964 [locus_tag=PC41400_RS15055] [protein=macro domain-containing protein] [protein_id=WP_042227525.1] [location=3284208..3284654] [gbkey=CDS]
MGVKIVQGNLMEANEDIIGHQVNCMGVMGAGLAKLIKRDYPEAFKQYKLSCGNKGELLGRCQIVSTQCEKYIANLFGQYG
YGVQKRHTDYIALRGALNELKSLAMKSNQSIALPYNIGCGLAGGDWDIVEQIIEEVFIDYDATLYKFE
>lcl|NZ_CP026520.1_prot_WP_129112496.1_2965 [locus_tag=PC41400_RS15060] [protein=hypothetical protein] [protein_id=WP_129112496.1] [location=3284669..3285040] [gbkey=CDS]
MIDLFLDDLRPCPKGFHIARTVEEAVSICESNNIRRLSLDHDLGGVEGNYTLTGYDFCKWLVEKWVLGDYSALPEAIFLH
TSNPVGRDNMRQILHRYNPGEVKVYNSPMPEYNLATGGFVEGF
>lcl|NZ_CP026520.1_prot_WP_053228649.1_2966 [gene=dnaE] [locus_tag=PC41400_RS15065] [protein=DNA polymerase III subunit alpha] [protein_id=WP_053228649.1] [location=3285381..3288530] [gbkey=CDS]
MLRDNYTVYHCHSDLSNPTTSMSMDSTTKFEQYLKVANELEMNSFCFSEHGSIMNWIKKKQTVEKYGMKYIHANEIYVTE
YIDREKGLIRDNMHYMLIAKNEYGVKELNKLTTLSNNRKDGHYYFNPRLSLDEVINTSDNIIMTTACLASPLWRALQKNN
AAMLNKLLEFLTRNKHRVFLEIQYHLHPEQIKFNQWLFDFSKSTGIPLIAGTDTHSLNQDHAAARKIMMKAKGATYGDED
LFDLTFKSYEELVDKFEEQSAIPRHAYLEAIHNTNVMADMVEEFTLDSSPKYPKLYEDSEKVFKEKINEGIVARGTNKLP
SEQKKRYYERVREEFDTYKKLGTIDYMLLQKNIIDYCYTKEIYPGYGRGSVNGSLIAYLLRITEMDSIKHKLNFFRFLNP
ERISLADIDIDFPPSRRQEVIDYVASIPNIHFSEIITFNTVALKGAIREVGRALQLDASLVDETAKAVFKDEEKKDCIPD
SYRKKYPELFKYVDLVSGVIISVGSHPSGYVVSPVNLDDSIGLCYTKESKYPVSQVNMKELDSCNFVKLDILGLDNIEII
NETCRLAGIERLVPDNMNVNDEVVWESMKDSGLGIFQWESESAHRYYKELFKQETIKKIREVNPEISLIELFSIGNGAIR
PSGSSYRESLAAGIFKDHGHKELNNSLNNTHGYLISQEQIMNFLVDFCGYSMAESDTVRRGLSKKEGTEQYIPLIRDRFV
SNMREKFSEDEEHSKAIIEPFLQVILDAQYYGFSDNHSNPYSHIGYGNGYLRHYYSLEFLTVMLNIAENLDKVGKIAEYA
NSIGVSIKPIMFRKSVSEYMMAKEENAIYKGIKSIKYLNEQIAKELYELRDNTYESFCDLLVDILENTTCDSRQLQILIR
LNFFKEFGENKELISIYDEFVGGKDAYKKTLKDITKLKRLNMKKKREAEIRSSLKEKRYPIHETVLYEKELMGYAGIRYP
NAIDWYCIVDMDKKYTPRLTIYDLMRGIEKQIKVDKKKFYSNGNDNFKIGDLLHLSQTEMRPKSKLVDGKWVKSADQFDE
WLVSCKIKK
>lcl|NZ_CP026520.1_prot_WP_164977129.1_2967 [locus_tag=PC41400_RS29820] [protein=hypothetical protein] [protein_id=WP_164977129.1] [location=3288829..3288981] [gbkey=CDS]
MNKKNEGKICEQNYKESVNKTEHFFLRLKDTAKWLRGGLKSWQLNKDHFN
>lcl|NZ_CP026520.1_prot_WP_042227523.1_2968 [locus_tag=PC41400_RS15070] [protein=hypothetical protein] [protein_id=WP_042227523.1] [location=3289306..3289518] [gbkey=CDS]
MENKFNINKTDLPLVLTAEQIIQVLGIGRRAGYELLKAPPFPVRRLGKRGIIKASRDALFNWLESKESGA
>lcl|NZ_CP026520.1_prot_WP_042227521.1_2969 [locus_tag=PC41400_RS15075] [protein=DMT family transporter] [protein_id=WP_042227521.1] [location=3290086..3291084] [gbkey=CDS]
MENALGLNGKSVLTKGKGINGIWYVAMGATLWGLDPLFRILLLKSLTSAQIVFIEHLLLACYALPVLIRYREQLTGKLTL
AVIGALLFISWGGSAIATILFTSAFAHGSANAVLLLQKLQPLFAITLARVLLKEALPARFFKYIFFALAGTYLLTFGFQS
PSLGLKDLGTLSCLLSILAAILWGGSTVMGKYLLQKNLDFPIVTSLRFLLAIPLLFIVLMASGDAWQLHKATGEWALIVV
NLLFQAFFPGLISMLLYYKGLSSTKAFLATIAELSFPAVGVLINWLVFDQALTIGQLIGFLLIWSTLWFMGRTPESKAAA
SGANEKLDASTV
>lcl|NZ_CP026520.1_prot_WP_241688661.1_2970 [locus_tag=PC41400_RS15080] [protein=DUF1835 domain-containing protein] [protein_id=WP_241688661.1] [location=3291293..3292099] [gbkey=CDS]
MKRPDMLDEIHALHPNEAKGLLLQMLEQARRYDEHSEKEHGFQKQILEIWNGHVSRKRAENEGRASHVHIVMGLSDAGAL
KCALSSLGRRKESEILAFNDCFSEGPIRGLETEEGRKFRLEWMFERFKGYGYAQLTGIRHGSDLILEAIRGLPADKDITI
WTGDHAHEQIGMRLALAVLRDAPNTVYIGHVSQLYERIAHRFDPVPHSYAFGFLSTEAIEEIVRLHEEGKRPLPAQTREA
YIRDWEELSASDAKLHIWSDGFCRSNAA
>lcl|NZ_CP026520.1_prot_WP_241688662.1_2971 [locus_tag=PC41400_RS30320] [protein=DUF3658 domain-containing protein] [protein_id=WP_241688662.1] [location=3292089..3292295] [gbkey=CDS]
MQRDASDDEEGFVRASRVIMEALESWGGAVMVEFIEYRLWTLISDGRLRFKGVPAAMQSYGVALAESP
>lcl|NZ_CP026520.1_prot_WP_042227519.1_2972 [gene=motB] [locus_tag=PC41400_RS15085] [protein=flagellar motor protein MotB] [protein_id=WP_042227519.1] [location=complement(3292500..3293300)] [gbkey=CDS]
MSKKHRREHHEEHIDETWLIPYADLLTLLLALFIILFASSQVDSKKYDSIMRALNSAFTGGTGTFESSAKIPLNRDSSTA
LKKHDNPNPSESEALQKQYEAEQADLEKLKQKLDQYITDNKLGSELTTTLTDDQLMITIQDKALFASGSAVIKPESRLLA
QAMTDMLSQYPKYRIEVAGHADNVPINNKEFDSNWDLSSKRALNFMKILLENNAIDPSRFRSIGYGEYRPVDSNVTTEGR
SKNRRVEVSIMRGIKASEVKKTAEGE
>lcl|NZ_CP026520.1_prot_WP_042227517.1_2973 [gene=motA] [locus_tag=PC41400_RS15090] [protein=flagellar motor stator protein MotA] [protein_id=WP_042227517.1] [location=complement(3293293..3294090)] [gbkey=CDS]
MEKSSIIGIILAIIAVGVGMVLKGASLTALFNPAALLIIFVGTAATLFIGFSGEELKNFPKLMKIVFTEQKLMPKKDLIN
MVADWTALTRREGLLSLEEKVNDIDDPFLKSGMRMIIDANDQDFVREVLMEDIAAMEERHRNGAQIFTQAGTYAPTLGVL
GAVVGLIAALSNLNEIEKLGHLISAAFVATLLGIFSGYVLWHPIANKLKQKSKKEVTLKLMMVEGLLSIQAGISSSAVKQ
KLLVFVPSSQREDEDSSKQPGMDNE
>lcl|NZ_CP026520.1_prot_WP_042227515.1_2974 [locus_tag=PC41400_RS15095] [protein=VanZ family protein] [protein_id=WP_042227515.1] [location=3294534..3295616] [gbkey=CDS]
MFSFYLFPISYAFLTFPIAAVLFTLPFLIVQYRKYGYVHFYRALLLYLFLLYMMNAVFLIILPLPPSIHNAPPAVESYAQ
WIPFHFLEDILKETAVVADQPRTYLHLLKERAFLQVFFNVLLTVPFGIFLRYYFRTSWVKCLLFSIGLSLFFEITQVTGL
YGIYDYPYRLFDVDDIMMNTAGGMIGYVAAEWLSSLLPKIESLDDKVDLAAKRVTYTRRGLAFLFDWFVMSPILAILAVL
NVPLPYAAGVVLYFIVLPYFTNGQTLGKWIVRIRLTGKSDKITLYELFVRYGLLYLFVGGLNQLYIRAGIENLPDILMVI
ATFAMFLFNAWFAIHVLRCVFNRNRQLFYEKKSGTRHVIK
>lcl|NZ_CP026520.1_prot_WP_042227514.1_2975 [locus_tag=PC41400_RS15100] [protein=NUDIX domain-containing protein] [protein_id=WP_042227514.1] [location=3296025..3296543] [gbkey=CDS]
MLKPKTIEADWGGHLVKLTWIAAAAPPEKDVTSVHGVCFYEGKIVLVRVKTRGYNLPGGHVEAGETPGEALHRECLEEAC
AEGVIRPVGMIEVSHENNPNFVPGGKYPLVGYQVFYRMDITACHPFTRDNECTARIWVEPGELPHVIEDHELIICAVEAA
LDCGPVSRPLDD
>lcl|NZ_CP026520.1_prot_WP_042227512.1_2976 [locus_tag=PC41400_RS15105] [protein=SDR family oxidoreductase] [protein_id=WP_042227512.1] [location=3296566..3297360] [gbkey=CDS]
MPEKQNGQPPAVLVTGGSTGIGRGIALAFAAAGYRVAITHYQDGESAAAAAEEISRLSGVPCGVIEGDLRKLETAEATVP
EALRILGRLDVLVNNAGIGLYGRIQELPAENLDSLYGLNFRAPLLLMRDASRHMIERGIRGCILNITSSRAERAYPKDVA
YGGLKAGLKRAAEAAALDLAPYGIRVNCLAPGAVLVREPEPEPDKFAETVPLGRIGLPDDIGHAAVWLASGQASYVTGVS
LRIDGGLILPGMPEDGSASWNPGV
>lcl|NZ_CP026520.1_prot_WP_042227511.1_2977 [locus_tag=PC41400_RS15110] [protein=Ger(x)C family spore germination protein] [protein_id=WP_042227511.1] [location=complement(3297388..3298512)] [gbkey=CDS]
MKRAGRIGTVLLLLAVLPGCWDEDSLRNARLGYGAGYDLTPDGRLMQTMEVVDESKQPEQQGSAKNEVESGVGYSVRQTS
DIIRTKVTGDIRYFKYGFMLLGKSVAEKDLYPYLDVLYRDPRNPTSRVKIAVVDGSTNEMLRLKQIGNILIGEFITRKIE
SLEEMSIFPELSLETMLTLLLDPGQDFVLPYLKQVGKNVEATGVALFNGQRMTGSLNVEEAILYSLLSGSHKDTARFVRK
IKEGDRSNLENYITFDAGGKKSKRKLKIQVRDRKVDVYLDLNLPVRVTEFAEDHLHKKDTVNRLNKDLSALLTEDAEALI
EKLQESGCDAFGIGRRLMARHTKLWKSMDWLQEYRNVRFHPKVNVEIVGSGVFN
>lcl|NZ_CP026520.1_prot_WP_042227510.1_2978 [locus_tag=PC41400_RS15115] [protein=GerAB/ArcD/ProY family transporter] [protein_id=WP_042227510.1] [location=complement(3298509..3299606)] [gbkey=CDS]
MKIEPKDTISQGQLFLLIIKLQIGVGILSLPYRLHQLAKGGGWISVLIAGVFLQLILVMMWMLMKRFPSYSVFRIASVVT
GRYAGALIGAAYVAYFLLVGGVVMMGAFDIINRWILQNTPRWAVLFLFMLTGVYLIRGTFRTISRIFILFTFLIGPMMFL
IGYGVHRANFLYLLPLDEAGIWNIVVSSKETITAMYGFELILIIYPFVKGSHGGKLKAVSAGTWFVVLFYAFTVFTCMTV
FSPQQLNMIPEPVMYLLRSLPFGAMDRADLLFLPIWMISMVASMTGYCYAAAFGIGHLLKHKDHKKAAPFAVIISCIIAY
LPQTKEQFELFSYIADASAYFFLIGFPVLLLLLSYALKKKEGDPA
>lcl|NZ_CP026520.1_prot_WP_241688663.1_2979 [locus_tag=PC41400_RS15120] [protein=spore germination protein] [protein_id=WP_241688663.1] [location=complement(3299596..3301131)] [gbkey=CDS]
MSLYRKLLRAARNSGPGENFHEPENSLQASDSIAINIGYLHHSLCHTEDMKQKPLPFAGTDGMLVYLESLVDQDKIQLHV
IQPLHENTGQEPDNLFNILETTRETALDNAVKALILGKCVYFLEGLQEFYILATEAVCKRSVEEPENEGVIRDPHYGFIE
VLNVNLFLIRKLIASPYLTVRYYKAGTHTQTRIAIVYMKNIANAELVQEVEERIRAISTDTVLTTGIIQEMTEDHPFSMF
PQHLATERPDRTASNIMEGRVAILTEGSPTVLIVPVTFFAFYQSPDDYNGRWIVSSFVRMIRMFSFLLAFTLPAIYIGTV
SFHPDILPLELVHTIKSSLERVPFPAIVEAMLLELIFELLREAGIRLPSRVGQTIGIVGGLVIGDAIVRAGLVSYTMIIV
VSLTAIASFLVPSHEMSSAVRILRFPLMIAASIFGYVGIAFGLMLLTIHLCKLESYGTPYFSPFAPLRIKDMKDVFVRFP
SWLLNARPHDANPKIMKQEEATSRSWKRDED
>lcl|NZ_CP026520.1_prot_WP_042227508.1_2980 [locus_tag=PC41400_RS15125] [protein=MerR family DNA-binding transcriptional regulator] [protein_id=WP_042227508.1] [location=complement(3301185..3301913)] [gbkey=CDS]
MNFRPIDIARKLRVSTSALRNYEAAGLVPPANRTPSGHRTYTELHAAYFECIQAMSPGFGMDITAEVLRRLQKGEWASAL
WCMTEAQTALFRDKRRADDTLAYLESGEAATAQEAQDPGGQSIGQVSAETGIPRSAIRHWEKTGLITANRNPANGYRTFN
RLQVRKIRLIHTLRQAVYSQELAELKKAIGLLDPDNLEAAIAIARDSLRYLERMNLHQLKGAYCVYRLCERLNVPLSPSH
RE
>lcl|NZ_CP026520.1_prot_WP_042227507.1_2981 [locus_tag=PC41400_RS15130] [protein=DUF5701 family protein] [protein_id=WP_042227507.1] [location=3302098..3302721] [gbkey=CDS]
MRFEEEFERQIRNLIDKGYPGFMGMEEAAFRELFAPLKERAAERMREDFKPGEGSLPFVLVVTTGTLPRDEMMRRVERSG
KNGFSVMEAEDLARFEPIEGVELPEGEAYLLLDVDTGSETLNRTPNEAMKSFAEWNRSPLTIEEGIAVITHLPDVLQKNH
GFSLCGSRCGDRRVSALWISGGKPKLGWCWAGNPHTWLGSASCGARI
>lcl|NZ_CP026520.1_prot_WP_042227505.1_2982 [locus_tag=PC41400_RS15135] [protein=nitroreductase family protein] [protein_id=WP_042227505.1] [location=3302907..3303509] [gbkey=CDS]
MSAESFLQAVRERRSIYSISGESPVSDDKIVEIVKEAVKHTPSAFNSQSARVLVLLGEQHHKLWDIVKDILRTIVKEEQF
AATEQKVNSFRSGHGSVVFFEDMSVIEGLQAQFPSYQDHFPNWSQQSSGMLQYVIWTALESEGLGATLQHYNPLIDEKVK
STWNLPESWKLIAQMPFGKPTAPAGDKEFKPVEDRVKLAK
>lcl|NZ_CP026520.1_prot_WP_042227504.1_2983 [locus_tag=PC41400_RS15140] [protein=FMN-dependent NADH-azoreductase] [protein_id=WP_042227504.1] [location=3303699..3304334] [gbkey=CDS]
MATVLYITAHPHDHQTSYSMAVGKEFIEAYREANPSDEVIALDLYNSDVPQIDADVFSGWGKLRSGSSFDQLSASEQAKV
QRLNDIVEQYIAADKYVYVSPMWNFSFPPVLKAYIDAICIAGKTFKYTENGPVGLLTDKKAIHIQASGGVYSEGPAAGFE
SGHSFLKKIMMFHGVPSFEGIFVEGMAAMPDKAPAIKEAAIAKAKEAAKRF
>lcl|NZ_CP026520.1_prot_WP_042227503.1_2984 [locus_tag=PC41400_RS15145] [protein=helix-turn-helix domain-containing protein] [protein_id=WP_042227503.1] [location=complement(3304566..3304946)] [gbkey=CDS]
MEIIPEQCKVVVALDMIVGKWKPLVIQHLISKDTLRFNELRRLMPDITQRMLTLHLRELEEQDIVRRVVYPQVPPKVEYS
ITEYGRTLEPILHSMHVWGAAHVEHMKQRNAEAAADAESRAASARP
>lcl|NZ_CP026520.1_prot_WP_042227501.1_2985 [locus_tag=PC41400_RS15150] [protein=BlaI/MecI/CopY family transcriptional regulator] [protein_id=WP_042227501.1] [location=3305096..3305512] [gbkey=CDS]
MKIHTFKTNESGLNRFFGPLEAKIMNILWDGPEMAIKDVQHSLEREKLVNFNTVMTVMNRLLDKGILSKRMDRRTSLYKP
VQSRDEFLECQSKELTHELMEEFGSSLVVNHMLDALEEVDPMLIEKLQQKILDLRKDK
>lcl|NZ_CP026520.1_prot_WP_042227500.1_2986 [locus_tag=PC41400_RS15155] [protein=M56 family metallopeptidase] [protein_id=WP_042227500.1] [location=3305514..3306362] [gbkey=CDS]
MWQSRSRFIFRSSVLIAGVVLFQMTMYALHVLFGIRIGFNVLNVCTEWMRSKGIFSIAHLLDVFVIGTFVYSMWQILRQF
VLSRRIFAKLDAYRHTKLSAELNRMYPEARGNIVVVVSDEPIALTMGFGKPAVVLSTGLLDMLDASEAEAVIHHELHHQQ
TGDPRSMFFLSLLASVLWYLPVLKWFHHQYKIAREILADNYAISRQGSSRDLGSALLKLLRRGGTVAMPFAYASFADTSI
NYRIRRIVDPESPISFRPPLTPTMISGYVLVVMTWLFVQILF
>lcl|NZ_CP026520.1_prot_WP_042227499.1_2987 [locus_tag=PC41400_RS15160] [protein=DoxX family protein] [protein_id=WP_042227499.1] [location=3306455..3306856] [gbkey=CDS]
MNSLNKWQYGAFILRIVLGLIFLVHGYQKFEGGLSGVAGFFGSLGLPGFMAYVVAVIELAGGILMLIGLGTRWIAAAFAV
IMLVAIFTAKLSLGFTGGYELELALLGMSLGLFFTGKQFLALENVFSRGKANA
>lcl|NZ_CP026520.1_prot_WP_129112443.1_2988 [locus_tag=PC41400_RS15165] [protein=IS3 family transposase] [protein_id=WP_129112443.1] [location=complement(3306993..3308324)] [gbkey=CDS]
MRKIRTEFTETETRLLESNPNVSRVSGRNISYAPAFKLAAVQANQAGEPPMEIFLKAGFSIELIGQRTPTESLYRWRETY
AKYGEVGLLKERRGTGSTGRRSKTESSAEMKLKQAEARIKLLEAENELLKKLEALERRNSKGLTPSERFQLINQTIRKHE
LRRLTRYLCQLAEVSASGYYRWCSAEESRQVRETADRYGFELIKEHFEALNRKAGALVIKMRLEKCNGVVMNHKKIRRIM
RKFGLVATIRQANPYRKMAKATQEHRTCPNLLERQFDQGEPEKVLLTDITYMRYGSGQWAYLSCVKDGATKQILAHYLSS
KLELSLVERTMTRLLKRLDGNIHPDAIFHSDQGMHYTHPKTRLLIAEAGFKHSMSRKGNCWDNASMETFFGHMKDELTYK
DCKSLKELRLRVNEYMTYYNSQRYQWTLKKMTPDEFRSHLLAA
>lcl|NZ_CP026520.1_prot_WP_042225646.1_2989 [locus_tag=PC41400_RS15170] [protein=5'-nucleotidase C-terminal domain-containing protein] [protein_id=WP_042225646.1] [location=3308653..3310947] [gbkey=CDS]
MKAKKAVSLTVSAMLLFGTVFTSGRAAAETAEPAKPAKADTEISILGTSDIHGRFMPWDYALDGPNTNGSLTQLFTMIKK
VRQENPNTVLVDAGDMIQDNSAELFNDQPKSPMMVAMNEMNYDAWVMGNHEFNFGLDVFKKITSQYKGQMLGGNIYKDNG
DRFLPAYTIIEKAGVKIGIIGMDTPMTAEFEKGTDHLDGIQFKNPVDETKKAVKELEGKVDVMIGLMHMGIENENGVPGT
GVADVANANPELAAIFAGHMHKLVKSETINGVLIAEPDKYGTHVSRIDLTFTEKDGKMVLKDKKAAALPVKDASGVTAVS
DPELEKTLKPFHDFARADANIEVAQLKGMNLVPKNEIEGIPSVQIQETPLSDFFSEVMLHYSKADVVAHQIDNDKARLDI
GPIKKKDIAYNYQFALGEITVYKVTGKDVKDYMEWAVGYFNSTRPGDVTVSFDKKRRASKYSTNDFFGGVKYDIDLTKPY
GSRITNLRKMDGTAIRPEDSLKLGMNAYRMDALKAKGGALEGRSFEQLWSSKDATAFGEMGGTIRSRAIAYLKEVKNGVY
EPQVQNNWKILGVDTTSPERADVVELINSGILPVPKTEDGKYTNVESINILDAVTKEEVAALAGKAKVDVSLFDGVQTKG
EFYRKLNAAVKAAGNGSGEPTAPEPKPKPEPKPEPKPEPKPEPKPEPKPEPKPEPKPQPAVKKQAQVTSYFLNVRTKASA
TAKIITAVPKGTILEVLGTSYGWVKVSYKGSIAYVYGEYVKMLP
>lcl|NZ_CP026520.1_prot_WP_164977130.1_2990 [locus_tag=PC41400_RS29825] [protein=hypothetical protein] [protein_id=WP_164977130.1] [location=3311257..3311400] [gbkey=CDS]
MEVVKSAAFDKRTYYRAIPDGIKGSGQPRSAHPPGTNNGWSANHLVF
>lcl|NZ_CP026520.1_prot_WP_042225647.1_2991 [locus_tag=PC41400_RS15175] [protein=glycerate kinase] [protein_id=WP_042225647.1] [location=3311455..3312597] [gbkey=CDS]
MKIVIAPDSYKGSLSAKEAGLAIERGVLGAGIGNFTTSVIPMADGGEGTLECLMESTKGRMVRAVVKDPLGRDISAVFGI
LGDGRTGIIEMAKSSGLYLLEESERNPLLTSTYGFGQLIAAALDEGCRHFILGLGGSATNDGGAGMLQALGVELRDSGGR
PLPAGAGGGALGEVAGITANGLDRRLADCTFTLACDVTNPFTGPRGASAVFGPQKGAAPDMVDRLESNLARFADVIRDCT
GTALHRLPGTGAAGGLAGGILAFLNAKLEPGFSIVSAVTGLEAAIKGADLVLTGEGRTDAQTAGGKTPAGVAGLAARHGV
PAVVLAGSVGDGIEELYGQGVTAVFSIVNGPMSLEEAMRRTPELLEQTAAQVMRIFAVRK
>lcl|NZ_CP026520.1_prot_WP_042225648.1_2992 [locus_tag=PC41400_RS15180] [protein=LysR family transcriptional regulator] [protein_id=WP_042225648.1] [location=complement(3312587..3313537)] [gbkey=CDS]
METKQLLTFKTAAENLNFTQTARMLNFAQSSVTAQIKALESELGAPLFERLGKRLVLTEAGRQFKLYADKIITLTEEAKQ
VVRGDEEQTGTLVIGAQESQCTYRLPALLKEFKARFPNVKLVFRPSNSDAHAREQLLEGVLDIAFIMDSCKPRDALHIET
LIREELKIVASPDHPLADAKNLSPKDLEHETFLLTEAGCSYRTILERTFASVSVYPLNKFEFASIEAIKQCVMSGLGIAF
LPGMVVAADLREGKLAELAWEAPEDALFTQIAWHKDKWMTPPLLAFIELTRKVFHAGETAPEHNPDAKPEEEPLTS
>lcl|NZ_CP026520.1_prot_WP_042225650.1_2993 [locus_tag=PC41400_RS15185] [protein=flavodoxin family protein] [protein_id=WP_042225650.1] [location=3313750..3314298] [gbkey=CDS]
MSIAVIYGGTRPDGNTEQLTEEAVKGLTVDRIHLNTYTIRAIIDGRHAEEGFPEMNDDYKEVIDRVLAADVLIFATPVYW
YSMSGIMKNFVDRWSQMMRDGDYPDFKKTMAAKKAYVIAVGGDLPEIKGLPMIQQFQYIFDFMGIRFEGYILGKGNKPGD
VLQDAEGMFAAERLRNKLAALN
>lcl|NZ_CP026520.1_prot_WP_042225651.1_2994 [locus_tag=PC41400_RS15190] [protein=arylamine N-acetyltransferase] [protein_id=WP_042225651.1] [location=complement(3314621..3315373)] [gbkey=CDS]
MNVEAYLKRMDTNRPAKPDLAALTELQLRHMLHVPFENLDVIRQVPITLDVASFYEKVIGRRRGGFCYELNGLFSTLLKE
LGFSVRLIAATVAKPDGTWGMPESHATILAELDRSYLVDVGFGDSARSPVPLDGTPVEDVSGVYRAAKVPEADGLYDLQR
AGEDGDWNALYRFSAAPRSLGSFAEACTFNQTSPESHFTQKLLATKATATGRVTISGDQLIETDGDVKSRQPLAPEELEK
TLNSRLGLFV
>lcl|NZ_CP026520.1_prot_WP_042225652.1_2995 [gene=ileS] [locus_tag=PC41400_RS15195] [protein=isoleucine--tRNA ligase] [protein_id=WP_042225652.1] [location=3315890..3318970] [gbkey=CDS]
MNKVDVKEKARTRELRVLNQWREDDTFRRSMENREGKPNFVFYEGPPTANGKPHIGHVLGRVIKDFIGRYKTMSGYRVVR
KAGWDTHGLPVELGVEKQLGISGKQEIEEYGVAKFVETCKNSVFEYEKQWRELTEAIAYWTDMEHPYVTLTNDYIESVWH
VLSTIHKKGLLYKGHRVSPYCPDCQTTLSSHEVAQGYEDVKDLSATVKFKSKAADEYFLAWTTTPWTLPANVALAVNKDL
DYVKVKHNGEVYVTAQNLAEKVFGDEAEVLSVHKGAEFVGTAYEPPFGYLELTKGHLVVDADYVSDASGTGIVHIAPAHG
EDDYRTTRQHGIDFVHVVKQDGTYAAEITDFAGRFVKDCDVDIVKKLSERGLLFSKERYEHSYPFCWRCKSPLLYYAMES
WFIRTTAVKDQLIANNNEVSWYPGHLREGRFGKFLEELVDWNISRNRYWGTPLNVWVCGDCGEEFAPGSHQELRDRAVDP
VDESLELHKPYVDDVKLRCSCGGVMERTPEVIDVWFDSGSMPFAQYHHPFGDEKTFNEQYPADMICEGIDQTRGWFFSLL
AVSTLYNGRAPYKAVISTGHVLDENGLKMSKSKGNVLDPWNIIEEFGTDAFRWALLSDSAPWNSKRFSKQIVAEAKSKIV
DTIHNTHAFFVLYADIDGFDPGSFERKPLRNELDRWILSRLNSTLGSVVKGLDANDFLNPAKQIEVFVDELSNWYIRRSR
DRFWGSGMSEDKVEAYQTLHEVLLTLARMIAPYAPLIAEDVYGNLGGKGSVHLADYPKADEAAVDTALESEMETARQIVE
LARSVRNETAIKTRQPLSELLISLDREFALDRFEAIIQEEINVKQVRLEHSDAGFVQFALKLNLKAAGKKYGKLVGPLQA
YLKNLSPEETKEAVSRGVLDTEVQGESVHLTLDELLVEKTAKQGFASASGYGLTVALNTEVTEALKQEGAVREVVRAVQD
YRKKLDLPIDRRITLVLDADPQLKEALLRYEQVLRDNVLVSAIRYENAEGMEEVALGESTLRMRVE
>lcl|NZ_CP026520.1_prot_WP_156972767.1_2996 [locus_tag=PC41400_RS29830] [protein=hypothetical protein] [protein_id=WP_156972767.1] [location=3319115..3319285] [gbkey=CDS]
MSEEPRNDVLGWFVGHLEKDGQEYVFALNVEADMEANGSAARKIAGEVLHGMRLWN
>lcl|NZ_CP026520.1_prot_WP_042225653.1_2997 [locus_tag=PC41400_RS15200] [protein=penicillin-binding transpeptidase domain-containing protein] [protein_id=WP_042225653.1] [location=3319373..3321403] [gbkey=CDS]
MIRNGKALAGIGGILLLLLLPACSQEETPQHAFEAYVSGWQALNYEAMYDKLSPASQTKISKDDFVKKYESVYKGIEMTD
LTVKPAFPEELQKPVKNKDEVSVPFHVELNTLAGPVKFDQTARLVKGKKEKDTVWQIDWSPSYLFPQLKDGDKVRVQSTE
AVRGEIVDRRGNKLAYNGPAKQIGVVPGKLGDAAEQTKAKLANKLGMKVEEINQKLSASWVKPESFVPLAYIQDGETEAY
TKLPGVTFSSKEIRTYPFDEAAAHLTGYIGQISAEELKKLQSEGYRTGDVIGKAGLEQIYEKRLKGQPGYKIFITGADGT
EKETLAEKVMVPGQTVTLSIDANLQKLVYEQMKKDAGSAAAIQPLTGEVLALVSSPSYNPNVFVKGISGTQYKQLNEDPK
KPFLNRFTKTFSPGSAFKMLTAAVSIDGGVLNPDEQKTIKGLRWTKDASWGSYYVTRVHDSSPVNLQSALTNSDNIYFAQ
AALELGKDKFVEASAKFGINEKLPLPYAFDTSQLYNKQIANDIQLADSGYGQGEVTMTSLHTALTYSALVNQGNIIYPSL
LKEDKGKEKYWKPSAMKPETAELLKKDLVEAVKDPRGVGHGAAVPGKLIAGKTGTAELKKAKGEEGQENGWFVGFDAADP
QLLVSMMIEDVKGRGGSGYITSRVKAVFQQYANGIN
>lcl|NZ_CP026520.1_prot_WP_042226245.1_2998 [locus_tag=PC41400_RS15205] [protein=hypothetical protein] [protein_id=WP_042226245.1] [location=3321563..3321817] [gbkey=CDS]
MDAKEMEEQLVRKLTEGEMKEEDREESARKRLMAKTEIRIQATTDPVVEETRKYREMAEELDGRYDRYTHLLEAKGPDSS
KDQS
>lcl|NZ_CP026520.1_prot_WP_042225655.1_2999 [locus_tag=PC41400_RS15210] [protein=prenyltransferase/squalene oxidase repeat-containing protein] [protein_id=WP_042225655.1] [location=complement(3321854..3323803)] [gbkey=CDS]
MHVMREVAHAIQSLSEWLLGKQEEDGTWKFCFDNGISTSAYLIILLRTLETENSDDEALIRRLHDSIIRTQQPDGSWKLY
ADESDGNLAASVEAYYALLFSGYSDDSSPTLVNARAFIRSRGGITGVTNVLTRVMLAATGQIPWPAAYSVPLELLLLPAS
APLSFFDLSAYARVHLAPVLLMADRNFSVRTSRTPDLSALDARRDEDGKNGVSSESAPGSSAQPDRGPGELLAAIQAGIS
QLAALPGQLHESAVKKAERFMLERIEPDGTLYSYASSTCLLVFALLSLGYERRHPTIVQAVKGLKAMLCVSEGRLLMQNA
PPTVWDTALIAHALQEAGVRPEAPEIRKAASYLLAKQQRKIGDWGRKVSHPVPGGWGFSPSNTRNPDVDDTTAALRAVKF
LRAEGAAGREAWNRGLYWIMSMQNHDGGWAAFEKDTDDKLLTLLPMEGAKHAAIDPSTADLTGRTLEFLGSAAGLDVKHD
WIRRGADWLTANQEKDGSWYGRWGVCYLYGTWAALTGLAAVGLKPSHPAVAKGMSWLLSVQNPDGGWGESCRSDISGRYK
SLGASTPSQTAWALDALIAVHPTPPVAIDRGIQRLLAALGERDWTSFYPTGAGLPGSFYNTYESYRYIWPLLALSHYRNK
YGEKAVKPE
>lcl|NZ_CP026520.1_prot_WP_206735002.1_3000 [locus_tag=PC41400_RS30085] [protein=YjcZ family sporulation protein] [protein_id=WP_206735002.1] [location=complement(3324023..3324151)] [gbkey=CDS]
MGYVGGVGTGAGCGVGGGLFTSTAVILVLFILLVIVSRTFLY
>lcl|NZ_CP026520.1_prot_WP_042225656.1_3001 [locus_tag=PC41400_RS15215] [protein=polysaccharide deacetylase family protein] [protein_id=WP_042225656.1] [location=3324416..3325582] [gbkey=CDS]
MKKALKIFLSLLLVAGSIYLVRQSAPRFTYKDQVAVLLYHHISDTDTSNVTITPALFRSQLTFLKKENYHFITLQQFKSF
MAGGNVPENAVLVTFDDGYESLYDQAYPVLKELQVPAVNFEITGTLADPRARKLHFLDEQQLRTIATDRDANISCQCHSD
SMHDKQNGKPLLTTNIVVDGKQETEEQYKKRVVNDALDCKKKLKKAGEEEVDSYAYPYGYYDARATSLLQSVGYKYGFTV
LSEMTTRADDPMQISRITAGSPDITPQVLNRTIQRNVMDFNQDYNYVPLRDTLKDLGGTMTMNKDRTISFYLHGKPYKLD
PESGTIETAGKQIPLKHPLLLKSRKAYIHFKDLENVLGVQIAHNPLRNSFFERLTPAKARNNETLTKS
>lcl|NZ_CP026520.1_prot_WP_042225658.1_3002 [locus_tag=PC41400_RS15220] [protein=ABC transporter permease subunit] [protein_id=WP_042225658.1] [location=3326180..3327142] [gbkey=CDS]
MPSSNSSRTPAAFQSKRGAAGWIRLFYRQRQLQTMALLGVAWILIFNYIPMYGILIAFKNFSIIKPVSEAPWVGFEHFRA
FLEDENLLSVIRNTLGMSLLKLVIGFPLPILFALFLNELRSVRFKKTVQTISYLPHFLSWVILGGILSTWLADTGLINKL
LEGLGILREPVAYLAEPAYFWGIVVSTDIWKELGWSAIIYLAAMAGVSPELYEAATMDGAGRFQKMWAITLPSIKGTITI
LFILAISGVLNSNFDQILVLRNSLNESASNVVDVYVYQMGIANSRFSYATAVGLIKSVIALGLLLGANYVTKRFNQTSLF
>lcl|NZ_CP026520.1_prot_WP_042225660.1_3003 [locus_tag=PC41400_RS15225] [protein=carbohydrate ABC transporter permease] [protein_id=WP_042225660.1] [location=3327160..3328047] [gbkey=CDS]
MILFKRKTAGEAIFDGLNGLVMLLVCFVTLYPIWYVLVHSFNNGTDAMQGGIYWWPRVPSLDNYRAVFENTGIMKAMGIT
VAKTVIGTAAHVFFTSMVAYAFSRRELVGRRAYMMLGTVTLFFSGGLIPTYLLIRDLGLLDSFWVYILPAMFNFFDLIIF
LAFFREIPAGLEEAAKIDGAGDFTIFCRVVLPVSMPVVATIALFHGVYQWNDYFTGMIYINNSDLQPIQTFLYRVVAQSS
SNQMLAAAPGGMAKNVTSQSIKLATMVVTTLPIVLAYPFLQKYFVKGFMIGSIKG
>lcl|NZ_CP026520.1_prot_WP_042225662.1_3004 [locus_tag=PC41400_RS15230] [protein=extracellular solute-binding protein] [protein_id=WP_042225662.1] [location=3328088..3329755] [gbkey=CDS]
MRKTSRLIPLVLSGVLALTAAACTQSGPGGETAKPDSTAAEVKIDPNAPGWKGDTSPVTFDWYLNFSWFGSKWGADPTSQ
YITKKTGVNLNLIVPAGNENEKLNTMIASGKLPDFITLGWYEDGVKKMIEGGMVLPLNKLADQYDPYFVKVADPAKVSWY
TQPDGNVYGYPNAAASPKDYETSGDKQTSNQTFLVRKDMYEALGKPDMRTPEGFLGALKAAKEKFPEVGGQPLIPLGLHE
FGDTGNYSLEDYNQRSSILQNFLAVPMEKDGKLYDRTTDPEYVKWLKTFRKANEEGLLVKDIFIDKRPQMEEKIAQGRYF
AMMYQRTDFAAQQNVLYAKDPNSVYIAVDGPGNAKQDPPVLSGGTISGWTVTLISKDVKDKARAIRFLTYMLSEEGQKDF
YLGEKGVSYDTIDGKDQFKPEALALLNKDRTAYDKRYGSAYTFWMLMDTTMNLKWAPPAVEPFKQLEDWTKGKTKSFSQF
DQVNPTGTSEEGIIAGKISALWGKTLPKLLLAKSDGEFDQIFAKFLKDRENAGIAKVEAYRQKVYEDNLKKLKQF
>lcl|NZ_CP026520.1_prot_WP_241688899.1_3005 [locus_tag=PC41400_RS15235] [protein=sensor histidine kinase] [protein_id=WP_241688899.1] [location=3329903..3331783] [gbkey=CDS]
MTSRLSGWKARTIQAAGAVPLQTRLIVSYIFIMLIPVVLLSMYLFNSFYESTIREIVKSNESMLEVEKTNIYNHMETMER
TAQLIVSDTAVAEYLQSNRNLEVNELLEYRRNAYDNLLRLQFNNPNLAHIRLFTSNANVKEIWPVFFSESRITGQPWYKQ
VKESPGTVQWIFGHSDPDLTRDTSIPSMDERGKVSLLREVSFVKGQHEAIVQVDMFFDLFFPKTFSRVQDGQSQMMIVDG
NGTVHTLYDTAFFRKLAPSGFLDRLNRDASPGGSFEFTSKGTRYLCVYAKLDRLGAFMLNIVSLEEPFSDIAKTRNNLIL
AMTGLIALLAVTTYFLLSLILKKLHVLQNSMKRVRQGNFHFDIDIQGGGEVGELAHHFRLMLRKINELIVEAVNKQAATK
EAELNSLKNQIDSHFMYNTLENLKMMAEIREQYALSDALTSLGGMMRYNLKWTSDYVRLADEIQHIGNYIAVMNIRYDDK
LRLQLEIPPEFWDQKLPKMSLQPIVENSVKHGIRSSGGEDVLVITVKARELNGRMSISVHDDGRGMSLHQLEQLRLTMNM
EDEAFRHRHSAEELREREGSGIGLRNVNQRVRLYYGREYGLQVTSREGAWTEVVLSLPLLSLTGGG
>lcl|NZ_CP026520.1_prot_WP_042225664.1_3006 [locus_tag=PC41400_RS15240] [protein=response regulator] [protein_id=WP_042225664.1] [location=3331790..3333373] [gbkey=CDS]
MRKLLIVDDEKNIRLGVSSMIRREFAGTYEIELASDGEEALRIVRDEGVHIVITDIRMPGMDGIALLRELQTLPVKPLVF
ILSGHDDFAYAKEAIRNGVKGYLLKPIMREELYEALRQAEKELDRQQARQEAQPGKFSTGSGELESFHSSQLNYILMHPE
MDAAEIRLRLATVGFTDNGSVWRLGLLRFPPDLRPRELDEGLTRLERLIPPNLQPLLNSASTFVFMDKDRSLVLLAPGDG
VPRLLRAAVPAGGRPLLRAGVSREKSDAALFKEAYREALRALRSTFIASAGGSVQLVEELEEKRDRGEIPREHIRQISNK
LGTGRLGEIKALLAHALDARTAREEGIAYVEAVSSLLNKDVFDQVFHTYGEQSVEILRIYKKAGTIDNFEHYHEYADCAE
RLVELLDEYVGRMKQVHVDHIEIKRAVKYMEENYARDLNMAMVSNYVSLNYTYFSKIFKEYTGDNFVGYLKKIRISRAKE
LLESSHNKVYEIAEQVGFENTKLFTRVFKEKEGITPQEYRAKRYMEG
>lcl|NZ_CP026520.1_prot_WP_255246816.1_3007 [locus_tag=PC41400_RS30595] [protein=hypothetical protein] [protein_id=WP_255246816.1] [location=3333437..3333565] [gbkey=CDS]
MGLKALANDKVVSADLNQGAVLIANRDSVGGAWEAFVITAAP
>lcl|NZ_CP026520.1_prot_WP_042225665.1_3008 [gene=serS] [locus_tag=PC41400_RS15245] [protein=serine--tRNA ligase] [protein_id=WP_042225665.1] [location=complement(3333695..3334981)] [gbkey=CDS]
MLDVKWIRAHQAEVEYAARHKGIEVSIPELLELDEARRSRLAEIEGLRQSRNALTEEIGMLLKQGNQPEAARKKQAVKDI
HLRLNDLEAAYRLTDQAYTAAMLRVPNPVSPDTPAGQSDKDNIELYRAGTLPDFAFEPKDHVALGNLHRMIDIPRGVKTA
GPRQYYLTGAGVLLHRAVQQLALDLLTEKGFMPVDVPLMARGDALLNTGYFPLGEDQTFKLAEEDKWLVGTSEVPLVSYY
SNEIVDLAEPVKMAAVSLCFRSEVGSAGRDVHGLYRVHQFAKVEQVVICENSLEKSDELLREITENAEELMRLLELPYRV
VAVCCGDMGQKTYKQVDIETWMPSRNAYGETHSSSNLLDFQARRSNIRYRNAEGRLTYAYTLNNTAVASPRILIPLLENH
QREDGSIRIPEALRKYMNGIEEIRAVQE
>lcl|NZ_CP026520.1_prot_WP_042225668.1_3009 [locus_tag=PC41400_RS15250] [protein=GTP-binding protein] [protein_id=WP_042225668.1] [location=3335521..3336729] [gbkey=CDS]
MDNKIPVTVLSGYLGAGKTTLLNHVLNNRDGLKVAVIVNDLSEVNVDADLIRDGGGISRTNESLVEMSNGCICCTLRDDL
LKEVERLAKENRFDYILIESTGVGEPLPVAQTFTYIDEEQDIDLTRYCRLDTMVTVVDAFRFWHDYASGETLLDRNQAVS
DDDNRDVVDLLIDQIEFCDVLVLNKCDRIDPDELLELERVLRSLQPKAKFIRSVMGEVAPAEILNTNLFNFEEASTSAGW
IREMEKEEHTPETDEYGISSFVYERALPFEPRRLKSWMEDWPAEIVRAKGIMWLATRNKFAQNLSQAGLSIRFGPAGYWV
AALPETEKQAILAEEGDSIPHWDPVYGDRLNKVVFIGLDLDQEEIVRSLDACLLTEEEMRSDWSRFADELPNLVGELNEA
SI
>lcl|NZ_CP026520.1_prot_WP_009673609.1_3010 [gene=rpmG] [locus_tag=PC41400_RS15255] [protein=50S ribosomal protein L33] [protein_id=WP_009673609.1] [location=3336767..3336916] [gbkey=CDS]
MRVTVTLACTETGDRNYTTTKNKKTQPNRLELRKYCPRLKRHTLHRETR
>lcl|NZ_CP026520.1_prot_WP_042225669.1_3011 [locus_tag=PC41400_RS15260] [protein=phospholipase D-like domain-containing protein] [protein_id=WP_042225669.1] [location=3337046..3338668] [gbkey=CDS]
MEKPKDTEQTSCEDAAAVREGIKNLINYLLSRGPVDKVIDGARQAVDFASANIPFEAAARYRQEIGGIFTSYEQLLSIPG
VGREVIEELGSYVRGLDLAKLNALLPQTTSGNRVEAYVNGPDCLRILLDEIGKAERYIHFSVMLFFNDQSGKKIAGALLD
ALKRGVKVRLMADFGITALGYDKNLDVGNLEEVIKKLEEAGGQVINTFKTCYGKEEWPAKRAELASRGVPESSLFLQDYV
QEQMTMGVNVVNHRKFMIVDGISSVVGSVNVGDQYLFDTPVLNDAEPIYAEGDREPRLGVPAREEEWHDGCFRIRGAAAL
TLNRIFSFQWTVLGGDVFDPEDSFYKPETDMRFGDEECTLYASFPGNPVNVIQQYYLSLLTYAGDETVIVNPYLIDQAFW
DELKSLDEQRSGCIALCNPLRVNDHPTNGSAVRSNMYEPFLKGVAFYDYSHTERFSHWKITYDKRSGCVFHGSYNINERS
ACHDFELGVLVKGRSFADQVKKMIDYDLSVSEKISGPETFFKYPALHPSTYWNKLTQNFT
>lcl|NZ_CP026520.1_prot_WP_042225671.1_3012 [locus_tag=PC41400_RS15265] [protein=lytic polysaccharide monooxygenase] [protein_id=WP_042225671.1] [location=complement(3338955..3340307)] [gbkey=CDS]
MTPISLRNPVLAKCLIASGLALALAFCLLLFTEKASAHGYIESPGSRGYLCAAGKNKNCGAIIYEPQSLEAYKGFPAAGP
ADGKIASASGLFPELDQQSATRWTKVNMSSGTNTFKWTIEANHATTSWKYYITKQNWDPNAPLSRASFDLQPFCSVNFKG
KQPGFTYTDTCNVPSRTGYQVILAVWEIADTENAFYNVIDANFGGGGPADTTAPTAPSNAAASNITAIGATISWSASTDN
VGVTAYKIYNGSTLVGTTANGTTLSYNAAGLTANTSYTFTVRATDAAGNDSAASNAVTFKTTAAAADTTAPTAPSGLQVV
GSPTSSSVSLKWGASTDNVGVTGYRVYQGATLVKSVSGTTLETVISGLSASTAYTYTVRAFDAAGNESADSNAVSATTSA
PPATAPWAPNTVYTAGTLVTYNGSTYECRQSHTSLVGWEPANVPTLWLKK
>lcl|NZ_CP026520.1_prot_WP_042225672.1_3013 [locus_tag=PC41400_RS15270] [protein=methyltransferase domain-containing protein] [protein_id=WP_042225672.1] [location=complement(3340678..3341616)] [gbkey=CDS]
MTHLSGRIRRQFDRSAAGLYDQNAAVQRYMTVQLVQLIRDELQPPPHSRLARIPANHQPVLEIGCGTGLLTERLAALLPP
GGEMTAADLSPAMLEAAKNRLDARSGRSDRGSAAPRESSATASASPEFPDTRPDSVRFLQADIEKWAADAPASSYGLIAS
SACFQWFTRPVQTLFHLRRLLKPGGKLMFATFGPATFTELHRSFDKAYRSMEVPPQRHGLSFHGGDQWTDMLRQAGFGEI
RTTRLVRTEYFPNVALFLKSVKALGASTTNALQVPGLSPRRLFGGMFEHYEKDFGTQGRIPVTYELLLIRAT
>lcl|NZ_CP026520.1_prot_WP_042225673.1_3014 [locus_tag=PC41400_RS15275] [protein=alpha/beta fold hydrolase] [protein_id=WP_042225673.1] [location=complement(3341579..3342625)] [gbkey=CDS]
MKRAPYQMPGGGVHPVMERGAEHGEKLSEECSTNSAARGLGHNTTHGDRHVAEQSAEQRAKHSTPIDTEFGLQPGKYTER
PGAYSEKSGSILWLSGWSMPPAIFLDVRRELPEFRHFYADYGLAESPEAIAESVRRLALSLRAGEPHGEPCSEPHERAAP
APLLIAGWSLGGLLALRLASEGLAAGLVLFGATACFVRQPAEQQSGWAPASVQAMIPALRQDADRVLQQFRRQMFTKEER
RDGLGANLPEASGWSVPGLIAGLELLAAEDVRDVMPDIRCPALLIHGTHDKICPYAAALEILAGVPQAVLVEAAGCGHVP
FLQKERQTAETIRSWWNDSSERQNPAPV
>lcl|NZ_CP026520.1_prot_WP_042225674.1_3015 [gene=bioF] [locus_tag=PC41400_RS15280] [protein=8-amino-7-oxononanoate synthase] [protein_id=WP_042225674.1] [location=complement(3342622..3343809)] [gbkey=CDS]
MNTMSWIDNELRALEDESLERTLADTGVSRSLPGHLMRGGQTLLDLTSNDYLGLSRHPDIAEAMRGALRAKCGSGAAASR
LLTGNRPPYSELEAELAAWQKKEASLVFANGYMANLGTIRALVGRGDAVFSDRLNHASITDGIILSRAGHVRYRHNDVEH
LEHLLKKSDKARRRLIVTDSIFSMDGDSAPLHKLVELKHNYGAMLMVDEAHAEGVYGPSGEGLCHAMDVHREADVIMGTF
SKAFGVYGAHITGSRTLIRWLVNKARPLVYSTALPPSVAAGIFRALGIVQTAHDRRNHLQEISGRFRQKLAQAGLSVMPG
DSPIVPVIAGSSRDSLRLCEALERHGIAAPAIRPPTVPAGSARVRFSLSAAHSDAVLDKAAESIVSAAEEAGILA
>lcl|NZ_CP026520.1_prot_WP_042225675.1_3016 [gene=bioD] [locus_tag=PC41400_RS15285] [protein=dethiobiotin synthase] [protein_id=WP_042225675.1] [location=complement(3343806..3344645)] [gbkey=CDS]
MSGIRGIFVTGTDTGTGKTVVTAALAAALRSEGWNVGVWKPVQSGEPIGSGLSDAEKLAAGSGISDPPEEVASFTFRAPL
APSLAAKAEGVRLTMDMLYEAGQPLCSRYDAMLVEGAGGAAVPLTEDALLAEWIAFLRMPVLIVARAGLGTVNHTLLTVS
FLRQRGIDILGVVLNEGTRASGVAGVAGPPGVPGVPYAADVQNAPDGPDPSVAGNAALIETFGGVKVLGRFPPIGDGPGP
RLLAETVHTALDLAPVRSFLSKCLEPSPELSKKAPEVRI
>lcl|NZ_CP026520.1_prot_WP_042225676.1_3017 [gene=bioA] [locus_tag=PC41400_RS15290] [protein=adenosylmethionine--8-amino-7-oxononanoate transaminase] [protein_id=WP_042225676.1] [location=complement(3344642..3346003)] [gbkey=CDS]
MLTTYNQLAELNKAHLWHPFTQMKDYCAADPVIIERGDGVRLFDTEGKAYYDGFSSVWLNVHGHCVPELNQAIIRQLELV
AHSTLLGLSNIPAIELAAKLSEIAPDGLRKVFFSDSGATGVEIALKMAYQYARNKGMESKRTFITMKEAYHGDTIGAVSV
GAIPLYHEVFRPLLFEALTIPYPDSYRHPGGASGAMSETLETLRLLLEARADEIAGLIVEPVVQGASGIIVMPEGCLREI
ARLCRKHGVLLIADEVATGFGRTGAMFACDLEDVAPDLMVVGKGLTGGYLPVAATLATDEVYEAFYADYEEQKTFFHGHS
YTGNPLGCAVALENLRLMEERRVVESIAAKAKFVADKLSSLKTCPHVGDIRQQGLLVGIELVRDRRTKEPYAWEERVGVH
VCSRARELGLLTRPLGNVIVFIPPLASTREELGAMTDIIAASIRDVTEQGWRP
>lcl|NZ_CP026520.1_prot_WP_042225678.1_3018 [locus_tag=PC41400_RS15295] [protein=zinc-binding dehydrogenase] [protein_id=WP_042225678.1] [location=complement(3346173..3347171)] [gbkey=CDS]
MKAILHREKSGVDGLVYSAAEDKHPGYGEVKVKLIAAGLNHRDLFVINDRNAGDRPLIPGSDGAGIVTEIGEGVSPDLLN
LEVALNPTLNWKKADSLPEDLQIIGSPSDGTFADFIIVPAENAVAKPFHLTWEEAAVLGLSALTAYRALFTRGGLKAGQH
VVIPGIGGGVATIALQMAKAAGARVSVSSRSEHKLEMARRLGADFCFESGGSWKALFEETRADLVLDSIGPATFPNYFDV
LKPDGRIVSFGASSGDSVELPLRALFFPQLSVIGTSMGSSEEFHAMLRFVAEHNLKPVVHKVFPMRDAVQAFAYMKEGRQ
FGNLTLMNDDKA
>lcl|NZ_CP026520.1_prot_WP_042225681.1_3019 [locus_tag=PC41400_RS15300] [protein=LysR family transcriptional regulator] [protein_id=WP_042225681.1] [location=3347312..3348181] [gbkey=CDS]
MESGDLKIFRTVAREGSITKAAHILNYVQSNVTNRIQLLETELGTQLFYRSSKGMKLTAAGQNLLGYADKIVVLLEEAVK
SAQATDIPKGPLRLGSLETTAAIHLPQLLTRYHGLYPEVELSLITSDSHRLIRKIKDYELDGAFVYGPVSQEELECQPAF
EEELVLISEPKETDLAGLLKKPLLFFGSGCSHRDRVERLLAKEGVEARHMIEFGTLEAIIGGVSAGLGVSLLPRSSISRI
VQAREVYLHPIPEEFREINVVFIYRKDLFKTEAFKVFLDFLRSEGTGRE
>lcl|NZ_CP026520.1_prot_WP_042225685.1_3020 [locus_tag=PC41400_RS15305] [protein=lytic polysaccharide monooxygenase] [protein_id=WP_042225685.1] [location=complement(3348451..3349797)] [gbkey=CDS]
MSTVRISRSITTRILFAAGLSLILAFCMLLLAQKASAHGYVESPGSRAYLCKQGTNKDCGAVVYEPQSLEYPKGFPAAGP
ADGKIASGGGTFPKLDEQSATRWSKVNISSGTNTFTWKLTAAHATTSWKYYITKANWDPNSALTRNSFDLTPFCSVNYGG
KQPPFSYSDTCNVPARTGYQVILAVWEIADTANAFYNVIDVNFGGSNPSDTVAPTAPSGAAASNVTTTGATLTWGASTDN
VGVTGYQVYNGSTLVTTTSGSATSYNVTGLTPNTAYTFTVKAIDAAGNTSPASNPVTFTTGSAPVDNQAPTAPAGLTAGT
ATSSSVALSWNAATDNVGVTGYRIYKGTALAGTVSGSTLTYTVSGLTPSTAYTFTVRAIDAAGNESANSNAVSVTTASGP
AVAAWAVNTAYTTGTLVTYNGSTYECRQPHTSIAGWEPSNVPALWQLK
>lcl|NZ_CP026520.1_prot_WP_042225687.1_3021 [locus_tag=PC41400_RS15310] [protein=ABC transporter ATP-binding protein] [protein_id=WP_042225687.1] [location=3350392..3352107] [gbkey=CDS]
MLKRFFAYYRPYRGLFILDFGCAVIAGLLELGFPLAVNLFLDKLLPGQDWSLITWACIALLAIYLLNMVLNYVVTYWGHM
LGVNIETDMRKKLFDHIQKLSFRFFDNNKTGHLMTRITNDLNEIGEIAHHGPEDVFIAIMTLVGAFLLMLSIHTKLALLT
FIVIPLLAWFAIYFTKKMTHTYERLFGNVAEFNSRVENNISGIRVVQAFANEEYEKNLFKGTNQQYRQTKLLAYKLMAKS
MSISYMMMRFVSIFVMLCGTYFVIRGELSYGQFVAFLLLSNVFFRPIEKINAVLESYPKGIAGFKKYISIIDTEPDVADG
KDAVSVDKLKGDIVFDSVTFGYDDQKNILDKLNLNIRAGETIAFVGPSGAGKTTICSLLPRFYDCKSGAIRIDGMDVRDM
TLESLRNQIGIVQQDVYLFEGTIRQNIAYGKLRASEEEIWEAARQAQLEEFIRKQPDGMETMIGERGVKLSGGQKQRLAI
ARIFLKNPPILILDEATSALDTATELAIQQSLTELSKGRTTLVIAHRLATIKNADRIIVVTERGVVEEGSHRELVAAGGI
YSGLHKAQFGV
>lcl|NZ_CP026520.1_prot_WP_042225688.1_3022 [locus_tag=PC41400_RS15315] [protein=MFS transporter] [protein_id=WP_042225688.1] [location=complement(3352268..3353467)] [gbkey=CDS]
MRIVLWMGCAFYFLVGVVHVFIGSLSPHFYARYDIGAQEISVILSVQFIGFLIGVLFSPVLSKKVGPFVVITIGLGMVGA
ALAGYLWIDRWELLSLTGFLIGYGAGTLETTVGALIIAQGETGKAGMAKLEVFFGFGALLFPILIHLLAGEDVIPLVLYS
LLSTIVVLLAAWVSLISILRPKQRFPEPEEAVSGSEEIYPGGGLMRGEKRSLSLLILFAFVYAGIETSYANFLPSLMIVG
GFESYSLIGVSVFWASMVIGRTVLAVKGKKIPPLPMLQISAVCLILLLGVFPFLHDAILVLGVISLTGIAASPLFPVAIT
FAASVRQGYVDEVTSYFIAAASLGGSLISLLVGWSLALEELTVTLFLFAGLALLLWFISFRLRDAGNGIYKPAHKSMDR
>lcl|NZ_CP026520.1_prot_WP_042225689.1_3023 [locus_tag=PC41400_RS15320] [protein=Gfo/Idh/MocA family oxidoreductase] [protein_id=WP_042225689.1] [location=complement(3353503..3354558)] [gbkey=CDS]
MIKIGIIGAGNIAMTHAKALSTVKEAVLTGVHDIHQPAGERFASTFGGQAFERYEDLLDSSDGIIVASPNFKHKDHALQA
LAAGKHVLCEKPMAITLEEAQVMRDYTDKAHAIAMMGFNYRYLSFVHILMEFIRNGGHGGIISFNISFKKDSALRKKAFT
WRDSEASKQTSGALGDLGIHLIDMIWYLFRSDFRPESIRVKMATHVKEKENREVQVDDYSEILGQLHNQVFIHMVTSKST
VPEECGFGIEVIGHRKNFHYHTRNPLSYTVTEGISSTSFDLPPALLTDPAGEFHGWADSFRIEQMAWVDAILRKSRMGLP
SFADGHRSQATLNLFFETHAKDLSGALVMQA
>lcl|NZ_CP026520.1_prot_WP_042225690.1_3024 [locus_tag=PC41400_RS15325] [protein=HAD-IIB family hydrolase] [protein_id=WP_042225690.1] [location=complement(3354621..3355478)] [gbkey=CDS]
MLSEYKDGFIQLEHPVKPKHIVFCDFDETYFPHVLLEEHRRELEELEAFIERKSREGELMIGWVSGSKLEDILRKMSGGK
FRIFPHFIASDLGTELTYVTRERYGMRDEHWDRRLESSGFSEETVKSLIGELSQEYGVDLTPQTQLGNSRFKWNFYYRQR
DPMTDLHTIKHIRDLAGRYRVDVNINRCNPLAGDPEDCYDVDFIPLGTGKAEIVRFMLERYGVQPEHAFAFGDSGNDLHM
LRAVRHGYLVHNATEEAKGLHPLVSEGAYVKGILHVLKEHLQAGA
>lcl|NZ_CP026520.1_prot_WP_042225692.1_3025 [locus_tag=PC41400_RS15330] [protein=DegT/DnrJ/EryC1/StrS family aminotransferase] [protein_id=WP_042225692.1] [location=complement(3355462..3356793)] [gbkey=CDS]
MLTTKNKYWGTSKDYLHLLDCSKTGRDVEPDCFENHLVNSKLLEKLYISRKENTINSWIRSLFPALSEFPGKVDFLPLNR
LISAEEVEAILAVMKVVLPTGQFTSGPYVNRFEEILSGYLRKKYIIATSSGTDAAMVALLALGLRPGDEVIMPANSFAAT
ENAVLAVGGVPVFADIDPHTYTLDPASAEAAITPRTRFVLPVHLYGKKANMAGLRELADRYRLKIVEDACQAIGVTELGR
EGDALILSFNPYKNVGVCGKGGAIALDDEEAARRCMQFSYHGFEVNVKNRKVRDFGYNSRMDNLQAAIGIERMKHLSLNN
FKRTILADRYRQGLQSLADSQKIRLPEMSDDHVWHHFPIQILSADRDLVRRELMDKYSAETDIYYPVLSHRQNTPFVDAR
YKDAHLPQTEQTHSRLLHLPMYPHLSWEEQDHLMEGLRRVIGV
>lcl|NZ_CP026520.1_prot_WP_042225694.1_3026 [locus_tag=PC41400_RS15335] [protein=LacI family DNA-binding transcriptional regulator] [protein_id=WP_042225694.1] [location=3356895..3357887] [gbkey=CDS]
MPTIQEIAKLCKVSTSTVSRVLNNHPYVSEKIRREVLRVIEEMNYTPHFLARGLRTNSSKLLAVSVPKADHPFFAGLIKG
ISLQALESNYKVCVFQTFYCKENELELMKLLQNREVDGIILGALENLWDDIAPYLKRGPILMCNEFHETAQVPILCYNEF
ESGYTAVRHLIERGYRDIGFCFDTANSEAQNIRKKGYLAALADHNLPYREEWIFDGAFSILDGIRIMNRIHDLPEKKPAA
LFSGSDQVAAGLIKQAVNLGYSVPGDLAVIGYDNQPICRVTTPTITSIDIPVEQLGRKAVSRLLACLTEELPLKREVTLF
PTELVVREST
>lcl|NZ_CP026520.1_prot_WP_042225696.1_3027 [locus_tag=PC41400_RS15340] [protein=alpha/beta hydrolase] [protein_id=WP_042225696.1] [location=3358024..3358848] [gbkey=CDS]
MQRFSKMALFMVVILLLSATTAFAQTSKVDVGSGLELLIDVQGTSVPGKPTVVFDSGSGDDHTVWQTAYVQQQIAQSTLT
VSYDRAGLGQSDDDANIAKNGVTKATQLHTLLQNANIPGPYLFVSHSISGVFARTYAALYPSEVAGIIFVDSTHEDVVQE
FGEEFIRGDIEYSGEFTWSDWLTTDQQIRQLSAQDALRNTPITVLSANCHGVCDEQNPGSLIDETKWASLQNKIAALSDD
VVHTTVDSGHYIQLNKPQAVISAINAIVTRINLQ
>lcl|NZ_CP026520.1_prot_WP_042225698.1_3028 [locus_tag=PC41400_RS15345] [protein=SDR family oxidoreductase] [protein_id=WP_042225698.1] [location=complement(3359101..3359841)] [gbkey=CDS]
MFMQNKVAWVTGGGTGIGRATSLELARRGAIVVVSYSQSREDAESAVRMIAEEGGRAAAFKADVTRESEVNELAERIAAR
YGSIDLLVNNASVTRHIPMSDLDSATEEVWDELYAVNVKGMFTCAKAAAPRMKKNKSGAIVNVGSIAGLTGVGSSIPYAV
SKAAVHGLTKSLAQALAPEIRVNCVAPGAVATRWWQGREETMNALIGKLPLQRIATPEDIAQIICSLLEQEAVTGQILTA
DAGQTL
>lcl|NZ_CP026520.1_prot_WP_042225700.1_3029 [locus_tag=PC41400_RS15350] [protein=LysR family transcriptional regulator] [protein_id=WP_042225700.1] [location=3359976..3360863] [gbkey=CDS]
MDFKTLKSFQKIVEHGNFARAAEELAYSPSTVTTQIQKLESDLGVLLFEHGKKIALTEAGRLFYEQSIPILQNMEQLQKT
LGDLVSGEAGQVRVGVSEPTASHQFPGLIRRFTDMYPNIRISVEIAGTSVLTDKILKGELDFGLGSFPDSAAQLHFEPLF
TEEFLLLVPQDHALARAESESVTLGQLRESRLLITAQSCPYRRKLETALREEGSFPPDTMEIGSMSALKHYVAEGLGAAL
VPRMVLDPLPEGTIALPMEGSGINMTCGLICRPADLPLKRAGKNFYDFVLHEFTR
>lcl|NZ_CP026520.1_prot_WP_042225701.1_3030 [gene=hemA] [locus_tag=PC41400_RS15355] [protein=5-aminolevulinate synthase] [protein_id=WP_042225701.1] [location=3361334..3362551] [gbkey=CDS]
MYTEIAESKLNQLKESGQYREFVTINRLRTKYPMALVDNQGDHPVVVWCSNDYLGMSQHPVVIEAMHKAIELYGAGSGGS
RNIGGSHHYYAELEASLAQWHGKEAALVFPTGYGSNDATLQCLARVFPDSIIFSDALNHASIINGIVSSKTEKAVFRHND
VQHLEELLAAQPADRPKIIVFESVYSMDGDIAPIREIVRLAKKYKALTFLDEVHAIGMYGPRGAGIAAQLGVADSIDFIQ
GTMAKGIGIIGGYITGSAPLIDSVRSFASGFIFTTSLPPAVVAACYASVEHLKRSDAERSALHAKTGLLRNALAEAGIPV
MESSETHILPVLVGEAKKCKLAARRLLEKHHVYLQPINSPTVPVGTERFRVNVTPNHTEEHIYELVNALTEVFEYFDIKT
LALKA
>lcl|NZ_CP026520.1_prot_WP_042225703.1_3031 [locus_tag=PC41400_RS15360] [protein=GNAT family N-acetyltransferase] [protein_id=WP_042225703.1] [location=3362617..3363114] [gbkey=CDS]
MNTSLALRIRRASLEDIGELVELRKILLSRGEGHYVSRSAEEDLAWQSEYRDWLRANLKGNDRILVAAASMDDEERICAC
AIGIIDDRAPMKGCLNGKVGWIQTVVVHPDKRRHGLGEGIMNYVLSWFRANEVGKVTLQTTPSAKRLYENLGFTESGEDL
LIKEF
>lcl|NZ_CP026520.1_prot_WP_042225704.1_3032 [locus_tag=PC41400_RS15365] [protein=Gfo/Idh/MocA family oxidoreductase] [protein_id=WP_042225704.1] [location=3363136..3364215] [gbkey=CDS]
MNVLIIGLGYAGTRFSQAFSNLDPAVTGNEPVEIAYVNRSPKSNGYRYFSDIETAVKEFEPAVVVVSVNDEYHIDVIKQL
RGYRGFVICEKPLVHPQDDLDEVEASLKHTSGFALDLVERYSDASITLKQYVKDHNLRLVRANFYWGKDRINDHRPTSGV
ISEIIHPLDLVQWVSGGDAELELKDLQGIRSDFSISGTAVLDSAAITANLAGAVVTGYSSFVNILRKREVDLVFASPANK
LIFATMIFDTPVWDIDYLRIWEKTNIGERVIVDLKTEIDESRPELNTIRKLIRLVGDAAGFAASGAEPSQPFADLPSSLK
LQKLLNTIDADARTIGPVQYVVGSEREFFNDETNWERLG
>lcl|NZ_CP026520.1_prot_WP_042225706.1_3033 [locus_tag=PC41400_RS15370] [protein=MFS transporter] [protein_id=WP_042225706.1] [location=3364260..3365435] [gbkey=CDS]
MQKVLLLTVGMFALGFDAYVVAGLLPAIGSDFNISASQSGQAVSVFTLCYALAAPIFATILAGKPVRRILLLALVVFTLA
NAGTALAPTYTLLLLSRAIAGIGAGLYSPLAAAAAAALVSSEKRGRALGMTLGGMSMGTVIGVPVGLLIAEQIGWRGTLW
AVTALGVIAIIGITARFPNFPAAAPPSLRQRLAMMTNGRVAATVGITFITSVASLGLYTYVASVLQDLADIESITPYLWA
WGIGGVIGSFSIGTLIDRTGRPRLLMTVILALMALAVFVLPVGIHYALAAFAAFVVWGATGWASQAPQQHVLLVLQPEHG
PAAVALNSSANYLGSAVGSALGGAVLLAGLAPGQLPIVAGALIAAALLGQIWIVSRSQGFAKAGREEKIQS
>lcl|NZ_CP026520.1_prot_WP_042225707.1_3034 [locus_tag=PC41400_RS15375] [protein=MBL fold metallo-hydrolase] [protein_id=WP_042225707.1] [location=complement(3365582..3366403)] [gbkey=CDS]
MIQYQSEQVTVFQSAIYKTTSTVIVTDDLVLIADPNWLPREIEEIRQHVEAVRGDRPVYVLFTHGDFDHIIGYKAFPGAV
TIGSEGLRNHPDKEKKLALITSFDAGHYIDRGYPVEFPHVDIAAAGNGQAFQLGETRLTFYLAPGHTADGLMTVVEPLGV
LIAGDYLSDFELPFIYDSAKAYEGTLDTVRHIYGTHNLELLVPGHGEATEDVREMERRLELADRYLASLIAAVQADDGNA
LLALRDAHAYCSAFTDGCHDENVRIVRSEYVGG
>lcl|NZ_CP026520.1_prot_WP_042225710.1_3035 [locus_tag=PC41400_RS15380] [protein=MFS transporter] [protein_id=WP_042225710.1] [location=3366578..3367873] [gbkey=CDS]
MTNFLNEWKTQLLQYSRNIRLFFVFIFVWNTGLGMFGLVYNLYVRALGYDQTMVGQLVGMTAIASAIILIPAGIMNDRFG
PKRIVSAGLFFVLLALTARSLFEFKNALLVTSFLGGMSQAVVSATIIPFMANNSTPRQRVHLFSFNMALSMFASVAGNML
GGVLTDMFQFAFGLSEVLSLRTTLLIGVGTMALGLIPVLLYTSEDAGEKDERKDERAGPKPGWRQTASRHKSSLQIIALF
AFLSLLSSTAGGMIVPYLNVYFEDRFDASKTMIGFIVSLGQGATALAFLLGPAIARKIGEARAVVLLQLSSIPFLLITAY
SSNFTLASVGYLFRQALMNAANPFYSTIKMKYVDRSMRGLASSSGEAMFNLGWFLASPVSMGLVATYGSYYGYAYAFSIT
AVVYTVICILFHVFFGKERFKPAEEAEQVAG
>lcl|NZ_CP026520.1_prot_WP_042225712.1_3036 [locus_tag=PC41400_RS15385] [protein=ABC transporter substrate-binding protein] [protein_id=WP_042225712.1] [location=3368140..3369810] [gbkey=CDS]
MISSIRHVHKAVLIALILAIVSGCGGNAGNTGSAISPASENGTAGQPSAVQGGTFTYALAGSPSGLDPNVVPAALDYRVM
RSIYDSLVVQLPDQSIKPWLAEEWTVSSDGKEYTFKLRKDVKFHDGTPFNAAAVKANFDRIVNPATKSRFAVTLLGPYES
SEAVDEYTVKVRLKSPYSAFLSSLSQAFLGIVSPAAAEKYKDQLTKNPVGTGPFKFVSWTENSAVTLQRNPDYKWAPELA
ANKGAPYIDKLVFKIIPEEATRIGSVQSGQVSAAETVPPQNFVSLKKDPNVQLFQAESSGIPYTLMLNQDHAPWNELEAR
KAVQLSIDFDAIVKTLYLGAYTRAWSPLTPSVLGYNASLEGVQKPDPDEAGRLLDGLGWIKGADGIREKNGKKLTIRYID
ASPNREKRNDIAAMIQQQLKKAGIAVEVEILADTTPALMVKGTNDLVGVSNVSGDPDILRSFFHTNAIPKDGKWGHNHTH
KSDPQVDAWLEEGLKEQDPKKRAEIYKQVQAYLIQNAYGFPVYVFPYTVAAGKNVNGLKFDSLGYPLFYDVSLKKS
>lcl|NZ_CP026520.1_prot_WP_042225713.1_3037 [locus_tag=PC41400_RS15390] [protein=ABC transporter permease] [protein_id=WP_042225713.1] [location=3369827..3370753] [gbkey=CDS]
MIYAIARRLVSSLLVIGGASVLVFFILYLLPGDPVLIMLAGTSASAETAANLRHQLGLDQPVIRQFLVYAENLLRGDFGK
SLVNDDPVLPKILEQFPATALLTAAGTFLAVVTGIWLGVVSAIHRNSFIDFLARIVSLFGISMPTFWSGILLILVFSVRL
NWFPAVGSEGWESLVLPAAALGLVGSGFIVRMVRSSMLEVMNEPFITTLRSKGLSERVVMYRHALRSALIPALTMIGIQI
GELMAGAVVIETVFSRQGIGRLLAGAILSKDLPVVQGVVFFTAIVYVLVNVLVDLSYAFIDPRVRRAV
>lcl|NZ_CP026520.1_prot_WP_042225714.1_3038 [locus_tag=PC41400_RS15395] [protein=ABC transporter permease] [protein_id=WP_042225714.1] [location=3370797..3371705] [gbkey=CDS]
MKTAAEVNYVPEQLTRERSREPKQNRKIPGFVRQGDLPYYAALLIVSAIALSAVVPQWLAPYPPVEMNADALLQKPGAAH
WLGTDYYGRDMLSLLIYGARDSLLFGAGSVVIGGAAGMLIGSLAGYIGGFVDGMLMRIIDVLMAIPSLLLSLAIAAALGP
SLLNIVLAVSVSLVPKFARVVRSQVIGIKSRPYIAASRSIGAGGGHIFVKHVLPHVFSPLLVMGTIGIGSSILVGSALSF
LGLGVLKEIPDWGTLLSQGRGYLTVAWWNATFPGLAITVLVLAVNLVGDRWRDALDPKQHAK
>lcl|NZ_CP026520.1_prot_WP_042226247.1_3039 [locus_tag=PC41400_RS15400] [protein=ABC transporter ATP-binding protein] [protein_id=WP_042226247.1] [location=3371739..3373817] [gbkey=CDS]
MAKPVLEIERLKVEFTKENESLTALSEVSLHIRPGETLCLVGESGSGKTILSKAILRLVEYEGGRITHGTIRLGQEDVTR
RPQKGMGAIRGRKAAMVFQEPMAAFDPLFPVGKQIAEAYLRHNRSSYKEALEHAARLLGKVGIPEPGLRMKQYPGELSGG
MLQRAMIAMALSCGPDLLIADEPTTALDVTIQSQILNLLKEIQKEYGMSILFVTHDLGVAAEIADRIAVLYAGGIVEVGG
ADTLFRTPVHPYTKGLIGSVTGLETPRGSRLPSIEGSIPPLAEPPSGCRFHPRCPHAIQRCAQSAPPLEEIAGREAACWL
AGELRQEEDAQAAEASPDIVRERAESATAVIRSAAVSAPREENTAGEVSGDGVPLFSVHGLRKTYRLGGGFLRRPKGSIY
AVDDVTFTIREGETFGLVGESGSGKSTLGRTLLHLEPPTEGEVLFAGKELSAMSPLELRKARKQMQVIFQDPYGSLNPRW
TVGQCIGEPLAVHGTHRGKERSDKVAELLASVGLPPSAGNRYPHEFSGGQRQRIGIARAMALQPRFILADEAVSALDVSV
QAQILNLLQELRERENLTYLFIGHGLQAVRHMSDRIGVMYLGKLVETAPSDELFRHPAHPYTAALLASVPRRDRSGSRER
IRLAGEIPSPAHPPSGCRFHTRCPAATALCREVQPEWIEVGPGHAAACHYPL
>lcl|NZ_CP026520.1_prot_WP_042225715.1_3040 [locus_tag=PC41400_RS15405] [protein=GNAT family N-acetyltransferase] [protein_id=WP_042225715.1] [location=3373859..3374428] [gbkey=CDS]
MGEIFRLAGENDTGQLLTLIEGAYQSIRDLNIDFRATRADYDFVRTNSVENACYVLERDRRISATLSVKYMEEVSVYPFL
FMLAVHPAEQSRGVGSRLLRYVEEKIVRDTLLSPASTLATSRKHPWLLPFYERKGYQPFYERDLGTDDKLVFMIKNLRSE
SGTNGEDSAHSSGEAGAEPYSFPGDLKKI
>lcl|NZ_CP026520.1_prot_WP_042225717.1_3041 [locus_tag=PC41400_RS15410] [protein=ABC transporter substrate-binding protein] [protein_id=WP_042225717.1] [location=3374446..3376089] [gbkey=CDS]
MKKRHRITWLIVLAAFMTGLSACGAPAASPASSPNPGGEPGKGANTGEKTVYIGIVNAPITLSQINDQGDSASDNVIALL
NDSLLNLNEKFEFTPKLAESVETKDNQTFVVKLNPSAKWNDGKPFTTADVAFTLQTALHPKVETYFKLNFIEGLNNAGKL
EEGKTEISGLKIIDDRTFEVRTKTPVDPLLFKERFGTKVFFLPKHILKDVPPEQIATHPYFQKPEVTIGAFKFANFVKGQ
YAEVVKNPNYYLPPAKLDKIFIKVLPAANLVAQLQTGEIHMNSLPVGLIPITEYEKVKALPNVELTTSNPSIPPELFFNT
ANIGDARVRQAVAYAINRKLIVDQLLKGQGEVIDGGIPSYHPNFNKDIQTYPYDPEKAKKLLQEAKWDINKPIRFLIPVG
NKVREQAADILAQNLQAVGLKVEVQKYDFATLIQKVRKKEFDLTIFNRDFYIEPSSYFTLYKSDSASNYYSYNNPKVDEL
ITKGETESDPAKRRQIYNELQAVIHEDVPTLAVYSEKRLLAVSKDVLVGRPRDIGMYNNTNEWDLKK
>lcl|NZ_CP026520.1_prot_WP_042225719.1_3042 [locus_tag=PC41400_RS15415] [protein=amidohydrolase] [protein_id=WP_042225719.1] [location=3376105..3377517] [gbkey=CDS]
MGKRDLVEWLDSHAEKFEELALRIWEQPEVAYTETYASKLQAAVLREAGFTLRSGIGGAATAFVAEFGSGKPVIGILGEY
DALPGLSQKVSAEREALVPGGPGHGCGHNLLGTAGVAAAVALSRRIREEKLPGTIRYYGCPAEEVISGKTFMAREGVFDD
LDTALTWHPGSTNMPWNVPTSALTSVEFSFKGRASHAGSFPHLGRSALDAVELTNVGANYLREHVVDGSRIQYTITNGGQ
APNIVPEDASVWYFLRGSGRDHVDELLRRLQNIARGAALMTETEVTWEIKAGAYDLNVNHTLNDLLYRQRDEAGPLSFTD
EDIRLAAALADTLPPDIRESALKRGEELGLPAGELLPAGFVNVKPAPKTYGGGSTDVGDVSWITPVGQIITTCAPFGVQV
HTWQATASFGSPIGLKGMHLASKLLALAAYELLTDGGVLLAEARAEFLRATNGRRYVPGIPSDVQAPVLV
>lcl|NZ_CP026520.1_prot_WP_042225720.1_3043 [locus_tag=PC41400_RS15420] [protein=ABC transporter ATP-binding protein] [protein_id=WP_042225720.1] [location=3377657..3378667] [gbkey=CDS]
MLPVLEIHRLQTVFKTGQEEVTAVDEVSFQLLPGETVGIVGESGSGKSVTALSVMRLLGRSGRIKQGDILFQGSNLALSS
DLELRQLRGRDMSMIFQEPMSSLNPVFTIGEQLLEVIRRHMDFSRKEARAYAIEMLGKVGLPRAEVLMKSYPHALSGGMR
QRVMIAMALACKPKLLIADEPTTALDVTIQAQILELMKQLREESGAAILLITHDLGVVAEMADRVVVMYAGQIVEEADVF
TLFREPLHPYTRGLLGSMPRISGDGGQLEAIPGTVPSLNRMPEGCRFHARCAWASAVCTQEQPELTALEGGRRVRCWVSQ
VEGRLPEKPGRKEAAL
>lcl|NZ_CP026520.1_prot_WP_042225721.1_3044 [locus_tag=PC41400_RS15425] [protein=ABC transporter ATP-binding protein] [protein_id=WP_042225721.1] [location=3378664..3379689] [gbkey=CDS]
MSGRPEVPAADDAPLLQVEHLKTYYPVKSGIFSRTTGFIRAVDDVSLTIRKGETLGLVGESGCGKSTIGRSIVRLENPTS
GRILFRDEDITTKTRRQLKPVRTRLQIIFQDPYSSLNPRKRVSELLSEPFIAHKLADKKDIPARVGRLLELVGLPRGSGN
RFPHEFSGGQRQRIGIARAISFEPELIVCDEPVSALDVSIQAQILNLLGDLQRELKLTYLFIAHGIGAVKYISTRIAVMY
LGKIVEIGNKEELFRNPKHPYTRILLNAYPPPDPEARSAGRILIRGDVPSPEHPPSGCRFHTRCPYVRDLCRTAEPPLGA
EAHAAACHFPLDEPSGLPVLK
>lcl|NZ_CP026520.1_prot_WP_042225724.1_3045 [locus_tag=PC41400_RS15430] [protein=ABC transporter permease] [protein_id=WP_042225724.1] [location=3379734..3380687] [gbkey=CDS]
MLPYAIRRILIAVPVLFGITVINFFLINMAPGNPVDMLIDPTMPKELLDARKELLGLNDPLWMQYAKWLGGLLRGELGYS
FSSYAPVSTLIAERLGPTLGLAAASLALGLLIAVPVGIISAVRQNTKFDYFMTGLSFIGTSIPQFFFGLSLIYIFAVQLR
ILPTGGMYTLGAGGGAWDLIRHMILPVFVLGVFIAGKKVRYVRASMLDVLKQDYLRTARAKGLHPFIVTNKHALRNALIP
MITVIAMEIPLLLGGSILIEQIFQWPGIGQLTIQSILSRDYPTLMGLNLAAACIVLATNLLTDLLYSAADPRIRYNG
>lcl|NZ_CP026520.1_prot_WP_042225725.1_3046 [locus_tag=PC41400_RS15435] [protein=ABC transporter permease] [protein_id=WP_042225725.1] [location=3380700..3381614] [gbkey=CDS]
MALTEWKLGETNFRSDTAESEAEPDSPRPWRQALRKFRRHKLAAAGSVVTLLLIAAALLAPAIVPYDPKEVTGVFSAAPS
AEHWLGTDQVGRDVFSRLVYATRVSLIVGFASVFLYVTFGTLVGLVSGYFGGWLDMAIMRITDMFMAFPFLMVILVVVAV
LGSDMRTIILVLALFSWPGVARLVRGSVLTIKRLDYVRAGIALGYGTPRILLQHVLPNAAGPVIVQATFGVASAIMSEAG
LSFLGMGVQPPTASWGNMLTDAQSLTVLTEQPWLWIPPGAMILLVVLSINFVGDGLRDALDSQP
>lcl|NZ_CP026520.1_prot_WP_042225727.1_3047 [locus_tag=PC41400_RS15440] [protein=response regulator transcription factor] [protein_id=WP_042225727.1] [location=3381831..3382556] [gbkey=CDS]
MTGIHILIADDEKEIRELLKTYLERETYVADTAHNGEEALRLAEERTYSLIVLDLMMPKVDGIEVCRRLRQTSNVPILML
TAKDQEVDKLVGLSIGADDYITKPFSIHEVIARIKAHLRRYKVLGSENSENSENSPAAEKALTSGGLKLDLKTFTMSKDG
EEIALTAKEFELLKFFMTHPEQVFTKTQIFRQVWNSDYLEDDNTVMVHIRKLRKKIEEDPSDPKRIVTVWGIGYKFAGER
L
>lcl|NZ_CP026520.1_prot_WP_042225729.1_3048 [locus_tag=PC41400_RS15445] [protein=HAMP domain-containing sensor histidine kinase] [protein_id=WP_042225729.1] [location=3382553..3383608] [gbkey=CDS]
MKRDPVKLLGAIHLILLAALAGLTAVGTPSALTNGVLFAALFASAAGLLISRFRYTARLRDVNRQLARALAGNVNTRVLA
NDDPALNEVIFSVNELIGQLDKLQVYTIKTETGRKRFLSNISHDIRTPLTSIIGYIDALRDGVAGSEEQKRQYLDVLARK
SGVLKKLIDEVFLLAKLDADEIRLQPARLDLAETVREAVIEFLPELHQAGLELVLQLPEEACPVVADRVSLLRMLNNLIQ
NAVKHGGDGNKLGLELSFAQGKYCLTVWDNGAGIPAEELSLVFDRMYRTDASRSRAGGGSGLGLAITKALAEKHGGTLGA
DSEPGVRTAFYLTLPETGKGLQPAADILRNN
>lcl|NZ_CP026520.1_prot_WP_042225731.1_3049 [locus_tag=PC41400_RS15450] [protein=ABC transporter ATP-binding protein] [protein_id=WP_042225731.1] [location=3383691..3384611] [gbkey=CDS]
MSAIIRTTGLTKTYGDQKAVDRLDLTVGQGQIYGFLGRNGAGKTTTIRMLLGLIKPTEGQIELFGENLFKHKRNLLRRIG
SIVEHSGFYENLTGAENLLINARLMGVHKKTAIQDALEIVSLHQEPKKLVGQYSMGMKQRLGIARAVLHHPELLILDEPT
NGLDPIGIKEIRRLIKSLAEERKMTILISSHILPEIEQLADQIGIVHQGRLLEEVSFEELRKRNRKHLEFQVSDDNKAAM
LLEKELDIHDYEVRGEGKIRVYSHIGRQAALNRLFVENGIDVSRLTISEDRLEDYFIKLIGGGTIG
>lcl|NZ_CP026520.1_prot_WP_042225732.1_3050 [locus_tag=PC41400_RS15455] [protein=ABC transporter permease] [protein_id=WP_042225732.1] [location=3384604..3385338] [gbkey=CDS]
MVNLLYTELLKLKRAKMFLVSVIGAAAAPVLMFIGFLNMKSEKPDTSVTYEISFYNTNMYVLLLSGTLLYGVITAYLFNR
EYVEDTLKNLLTIPVSRTSLLFSKLLMLLIWIALLTLISWGFTFVFGLIGGFDGLTIDVILYSLGRYLIGAVLLFLLSTP
SIFVTFWFKNYVPTIIFTAIATMTNVALVDKKYSALFPWSAVHVVANKAYVPEYPVLYSVLAILAASIAGLAATIIYFNK
SDIQ
>lcl|NZ_CP026520.1_prot_WP_042225734.1_3051 [locus_tag=PC41400_RS15460] [protein=undecaprenyl-diphosphate phosphatase] [protein_id=WP_042225734.1] [location=3385354..3386181] [gbkey=CDS]
MNLWEAFAAFILGLVEGLTEFAPVSSTGHMIIVDDAWFQTKDLFTPQAANTFKVVIQLGSILAVVVLMRDRFMNLFGLKK
LPGQDEGGPRLNLLQVIVGLIPFGVLGVLFEDYIDEYLFSTRTVVIGLVLGAVLMLIAEWRRPVVPKTDTVDRLTYKQAL
GMGLFQCLAMWPGFSRSGSTISGGVLLGLSHRAAADFTFIMAVPIMAGASFLKLLKSWDSIPAEAIPFFAVGFVSAFLFG
LISIRFFLKLINRIKLVPFAIYRLVLAAVIYIVLL
>lcl|NZ_CP026520.1_prot_WP_042225736.1_3052 [locus_tag=PC41400_RS15465] [protein=alpha/beta fold hydrolase] [protein_id=WP_042225736.1] [location=3386288..3387247] [gbkey=CDS]
MPLIDLPLEKLKEYQGRNPRPTDFDEYWARALAELDSVDPQVELVPSDFKVPYAECFDLYFTGVKGARIHAKYVKPKSDG
KPHPAVLQFHGYTGHAGDWLERLPYAALGYSVAALDVRGQGGKSQDIGGIAGNTHNGHIIRGLDEHPDELLFRHVFLDTV
RLARIVMDFEEVDGSRVATMGGSQGGALALACAALEPRIRKVAANFPFLSDYKRVWEMDLAKDAYAELRTYFRFFDPQHK
REDEVFYKLGYIDIQHLASRITADILFGIGLMDTVCPPSTQYAVLNKLKAPVQLEIYPDFAHEGLPGMGDKQFMFLQQL
>lcl|NZ_CP026520.1_prot_WP_042225738.1_3053 [locus_tag=PC41400_RS15470] [protein=DUF3626 domain-containing protein] [protein_id=WP_042225738.1] [location=3387397..3388509] [gbkey=CDS]
MMELKRCQERALKSVESKAAAAEPHARKKAARILREAAYDVMSADDLCAAVLARSRVTLNFHPDRLIAGGLTVAEGLLRD
GEYRSQFETRISSGGLSAYPGGDRDRWERSLFAGAYQAEGAEPCDRPKYGALNVMRYADGASPRFGSCHFRLKAGVLDRC
TFTCGDSHAQPEHAGTRKVFMPVLAALLEEVRTTGSALASDVPGAGAFAARCLSAAEPDGKDGLPGPGASGGIGRALDAY
VEAQIHGAIRLESDVAALVADPSFQGTATGAQLDALSEAYGIPLYWHPGFRLAAGSVPEDFRGPVMPPLARRVVSSYGTE
GASPDAFDAAAIGRAAASLHRQPESWADFGTPDEVLQYLKQLWHVLVRYG
>lcl|NZ_CP026520.1_prot_WP_042225740.1_3054 [locus_tag=PC41400_RS15475] [protein=bifunctional 3-deoxy-7-phosphoheptulonate synthase/chorismate mutase] [protein_id=WP_042225740.1] [location=3388918..3389991] [gbkey=CDS]
MAQGNLDTLRERLDTINGQLLELLSERAEIVQEIGLEKERQGVPKFDPVREKKMLDELVALNKGPFSHETVRHLFKQIFK
ASLNLQEEGQKKHLLVSRKNKREDTIVTLGDGISIGNGKPVMIAGPCSVESYEQVRQVAAVLKEAGVTVMRGGAFKPRTS
PYDFQGLGIEGLKILKEVASEFGLKTISEIVDPAHIEIACDYIDVIQIGARNMQNFELLKAAGDVRVPVLLKRGLAATLD
EFLNAAEYIVSRGNTQVMLIERGIRTYEKATRNTLDISAVPILKQESHLPVLVDVTHSTGRKDILAPCAKAALAAGADGV
MVEVHPDPATALSDAAQQLNFEEFAQFYKTVTESGLF
>lcl|NZ_CP026520.1_prot_WP_164977131.1_3055 [locus_tag=PC41400_RS29835] [protein=hypothetical protein] [protein_id=WP_164977131.1] [location=complement(3390083..3390256)] [gbkey=CDS]
MKIYAVLSFNGESTENVCVTPDEEKAMALKPEDFEDCDALFLEIWEDGEKIDDYRLV
>lcl|NZ_CP026520.1_prot_WP_053228600.1_3056 [locus_tag=PC41400_RS15480] [protein=copper amine oxidase N-terminal domain-containing protein] [protein_id=WP_053228600.1] [location=complement(3390353..3391327)] [gbkey=CDS]
MNRKVKSALLVTLLAASVAMPGSVFAQEPVKVFIQNERVHFDVEPAIESGVTLVQFRPIFEELGLTVEWDSKTNTVTGRK
DGLSIVLQINNKNAAVNGETKVLEVAPANVDGNIVVPLRFIGEATGNQVNWNESARQIVIRSGPERGSGNSATVSDPYFN
WTWGMSPAEVQATLNQKPYIQYNGDKGGYDFVLYHNRFVEENDSQLAFKFNSSGLTEVAYLPVLSSTYDEIYDMYGEISG
TLSAFYRQDDEDYLIWSDDLAESFYKQIYGKDRRGMIEMAMGSENLTIFNQFEADTYKASLLFTNEGTLTEPEYKMGLIY
SRPK
>lcl|NZ_CP026520.1_prot_WP_042225742.1_3057 [locus_tag=PC41400_RS15485] [protein=RtcB family protein] [protein_id=WP_042225742.1] [location=3391759..3393183] [gbkey=CDS]
MKENYYRQVKLPSGTVHVYAGPSLYEALDYKVFEMANNNLQIPNGVYMSYTPDVHVGVGTCIGTTAVWNSEDGYVSPSIV
GSDIGCGMRVHLTNLHKHELKDLKLRKKLVRTIEKYLPVDAHTRGHFSDIRLETVVKKGLHGLPKKYIPDGYTPRKAKSL
THVESSRFGFDENSLNRLPENVWHRSHRQLGTLGGGNHFAEIQSVEIAEENRATAEAWGLFDGQVIVMIHSGSRAWGGSV
SQLCGSDIKRMMRAEGMGTADPNLLFASLDHPLAVQYTDLMYSALNYAVVNRHLIAYAVREAGKDVFGSKFELATLYDLM
HNYAWEEDTEDEGRRVFVHRKGATRALPANHPDNPEPYASTGHPALIPGSMGTSSYIMVGADGGKANYHSICHGAGRIRS
RSATKRLVTVDEFSASMRVGMEDEVVVNQRSLETILDESPQAYKDVDQIIESVVGAGLASVVAKCKPMAVIKGV
>lcl|NZ_CP026520.1_prot_WP_042225744.1_3058 [locus_tag=PC41400_RS15490] [protein=RNA 2'-phosphotransferase] [protein_id=WP_042225744.1] [location=3393189..3393743] [gbkey=CDS]
MPASKKDTRLSKLMSKMLRHTPGDFGLILDKEGFCRLKDFLAVLNRVSGFKDVRPEDIQRVVRECEKQRFEIVEDSIRAR
YGHSHERITYEAAEPPSVLYHGTNRKAVPLIMTDGLLPMSRQYVHLSEGLHFASLAGKRRGELVILEVDTALAAQKGVRF
YYAGNEVWLADRIPAVCCQPIADQ
>lcl|NZ_CP026520.1_prot_WP_042225746.1_3059 [locus_tag=PC41400_RS15495] [protein=MFS transporter] [protein_id=WP_042225746.1] [location=3393844..3395037] [gbkey=CDS]
MNHAQLAGKTESAKDRFPLALISLTCGAFAIGMTEFVIMGLLPNVAGDLNVSIPHAGQLITGYALGVAVGAPLLAVATHR
LPQKMLLCLLMVLFILGNTVAAIAPNYGLLMGARLLTALAHGTFFGVGSVIAANLVKPHRRAAAVSIMMAGLTIANIIGV
PFGTFIGQNLGWRASFGAVAVMGVISLIGILALIPVIPREESSSLVRQIRALAQPRLLLVLLTGAIGCSSLFTVFTYIAP
LLEQVTGYAEHSVTWILVLFGFGVTFGNILGGKLADWKLMPVMIGSFAVLALILAVFAYTVYLPAAAVVTIFIWGVAAFS
VLPGLQVKIMTLAKDAPALASTSNHSALNLGNAGGAFLGGMVINQAGLSTLPWAGAVLSLIGILLALWVFALDRKAA
>lcl|NZ_CP026520.1_prot_WP_042225747.1_3060 [locus_tag=PC41400_RS15500] [protein=retropepsin-like aspartic protease] [protein_id=WP_042225747.1] [location=complement(3395240..3395632)] [gbkey=CDS]
MQIDLIDGLLQTSMTICYEGQSLIIDNLVIDTGAAHSLLSSDVVSEIGIKFENGDKLLRSFGIGGDEYSFQKRVDQIQLG
NFIIEDFPIDCGVFHEDINYINGLIGLDILKTGNMIIDLHQMQLYPATSK
>lcl|NZ_CP026520.1_prot_WP_042225748.1_3061 [locus_tag=PC41400_RS15505] [protein=hypothetical protein] [protein_id=WP_042225748.1] [location=complement(3395633..3395875)] [gbkey=CDS]
MQWQEVRSIYPDQYVLLEILDSHTEDNIQFVDEVALVRAIQDPDEATKELLKCKNNNIVYHTGQEKISIELRKKPLYRSR
>lcl|NZ_CP026520.1_prot_WP_042225750.1_3062 [locus_tag=PC41400_RS15510] [protein=AraC family transcriptional regulator] [protein_id=WP_042225750.1] [location=3396186..3397820] [gbkey=CDS]
MSDSVTDNAKIRPGLSAAVADASYKIRQVETIKAGDGWEMDYRLIDQHVLIVVTAGEGALKRDGSLSRLEAENIHVCLPG
EVIGMKTEAGGLELTWVGFDLFYEDGTGEGCRKGTWNEPQVLPVHSLSHFSVLSEALSTYWHSADPFDRLRAQAAFLELI
HSISAHMRDESPLDARSALERTRHYIENHYQTNITIDELARMAEVSPKYYGDLFKKTYGLSVIDYLTEQRVNRAKQLMAL
SQSRLKDIAHVVGYQDEFYFSRKFKQEVGVSPTVYMKNRRKKIAVHHSHLIGQLLAIHHLPYAAPLHPKWTGHYYENYRS
DIPVHLSAYRYSEDWEHNIEKLRETKPDLILSMDTLNPGEKERLEQAGKVLYIPDRRADWREQLLLIAAHLDRSDEAETW
LGKYDRRVESLRGRLRKRMGEETFLLISLHKQEIRVCGTRSMGDVFFHELGMKPAFDYSRDQINVPILPEQLAELDADHI
LVNVCQETETLRRWQDLQATALWRDLHAVRQNRVYEIPSDPWREYSASGHERILGEVEKLFLRK
>lcl|NZ_CP026520.1_prot_WP_241688664.1_3063 [locus_tag=PC41400_RS30325] [protein=alpha/beta hydrolase-fold protein] [protein_id=WP_241688664.1] [location=3397989..3398453] [gbkey=CDS]
MESSRERGSLEQQVFQGRTLTVYTPPSYGLADKRYPVVYVNDPSDYMDNVTNLAVVRMEELFRERKLPELILVGLEPDER
LDEYTPWPAKALVESVKDFGGFGKKYLDLIVHELKPYIDGRYETLAGPAGTAMIGASLGGLISMFAAFEHPAVF
>lcl|NZ_CP026520.1_prot_WP_241688665.1_3064 [locus_tag=PC41400_RS30330] [protein=hypothetical protein] [protein_id=WP_241688665.1] [location=3398517..3398789] [gbkey=CDS]
MIADEGQRFYMYVGDQEGVGKKTIQSAMMARTRQAYALLADSGFNGAKLRLVIGEGGHHLVPVFVEQFPEALKWIFGMDT
NTFFPDTERG
>lcl|NZ_CP026520.1_prot_WP_042225752.1_3065 [locus_tag=PC41400_RS15520] [protein=ABC transporter substrate-binding protein] [protein_id=WP_042225752.1] [location=3398796..3399776] [gbkey=CDS]
MKKTWVAAGLVAVTLLSSACGAQKEAAKTGGAEATKSPAATAAGTNETRKITYLGKEYTVPSKAERIVVAGSFESMEDAL
VLGVKPIGANSVGGKFPEMFKSITDKAESVGEKTQPNLETILKLKPDVILGSTKFPPEMMEKMNKVKVTLPISHVSTDWE
ANLNVLAELTGQQDKAKEAIEKYKKEAAAAKEKLGTSLKDKKVLAIRVRAGSIAIYPAGVFFNPSLYTDLGLAVPNEVKE
AKAQQLISLEKLSEINPDYMFVQFSEDENKEQPKALEDLEKNPIWQSINAVKNKKMFVNVVDPLAQGGTSYSKIKFLEAA
LPKLTN
>lcl|NZ_CP026520.1_prot_WP_042225754.1_3066 [locus_tag=PC41400_RS15525] [protein=iron ABC transporter permease] [protein_id=WP_042225754.1] [location=3399805..3400812] [gbkey=CDS]
MKRSLAKPQWILWVSPVIIVFTIVLSVLYGAKQIHLQTVMDALFHFDPGNVDHQIIRTSRLPRVVGALLIGAFLAISGAL
MQGMTRNYLASPSIMGVTDGSAFVITLCMIFLPNSSSLDMIFYSLIGSALGVGIVFGLASLLPNGLTPVRMAIVGTVIGT
FLSSVSAALSMYFQKSQNISFWYNARLHQLEPKLILLCIPFAIVGLIAAFRIAKSITILSLGEEVAINLGQRTLLVKIVA
MGAVVILTGISVALAGKIGFVGLIIPHITRFLIGIDYKWVIPCAGVLGGIFLALADVLSRFLNYPYEMPIGVVTAIIGVP
FFLYLARTRGGEKHA
>lcl|NZ_CP026520.1_prot_WP_042225756.1_3067 [locus_tag=PC41400_RS15530] [protein=iron ABC transporter permease] [protein_id=WP_042225756.1] [location=3400805..3401827] [gbkey=CDS]
MPKQSFSRYMLVLLTGIGLIVIAAYTSLTNGSFDIMIGDVVNTLLRIDPSPENDLVILEFRLPRIVIAGLVGMGLAIAGT
VIQGITRNGLADPGILGINAGAGAAIVIFMFFFQGKLHGTGWWGIMTMPLFGLVGGLASAILIYLFSWKNGRIDSQRLIL
TGIAIGAGFSALALFLTLKMDAKDFEMAAVWTSGSIYNANWKYVVSILPWLIILIPLIRRSAYTLDLFQLEESSVKSLGV
NVERQKGILLLGSIGLVSACVSVSGGIGFIGLMSPHIAKRLVGVHHNRVLPVSAVIGMLLVIGSDFIAKTVFMPVELPVG
VMISIIGVPYFIYLLFKAKA
>lcl|NZ_CP026520.1_prot_WP_042225757.1_3068 [locus_tag=PC41400_RS15535] [protein=ABC transporter ATP-binding protein] [protein_id=WP_042225757.1] [location=3401843..3402697] [gbkey=CDS]
MSIIRVENLETCYEKNSVFRDLNFEVREGEVTTIIGPNGCGKSTLLKTIGRILKQKNGNVFFRGQNLNTIPTRKIAQHIA
LLPQNPVAPPELKVEELVSYGRYPHRKNLNRLSPQDKEVIEWALTVTNVAPFRDREIGSLSGGQRQKVWLAMAFAQDTEV
LLLDEPTTFLDMAHQLEVLQIVEKMNMERKQTIVMVLHDINHAARFSHRIVAMKSGQIIATGHPHEIITREVLNQVFHID
ARIMLDPENGAPVCYGYESYLRNKAASAAGSTEPQKEAVGEGVR
>lcl|NZ_CP026520.1_prot_WP_042225759.1_3069 [locus_tag=PC41400_RS15540] [protein=alpha/beta hydrolase] [protein_id=WP_042225759.1] [location=3402699..3403601] [gbkey=CDS]
MSTINREYAEPGGAAAGVLERVTMPRSEQRTMRSRAENREYRIFIAVPEGEAPPSGFPVVYVLDANSVFGTMVEAVRVQS
RAPERTGVIPAIVVGIGYPTDGPFHPARHYDFTREIRLEDLPFKLRAGERPEHGGAEAFMAFIEDELKPRIEETYPVDRR
RQTIFGHSLGGLFVLHMLYAKPEAFQCYVAGSPSIHWNKQVLLEEERAFASKLSASAGESTGETADIKLFLAIGELEKSH
HTGGGENAAELCERLSALSEYGMVTEFKEFEGEGHVSVLPPLISRAVRFALRPEPADRQV
>lcl|NZ_CP026520.1_prot_WP_042225761.1_3070 [gene=greA] [locus_tag=PC41400_RS15545] [protein=transcription elongation factor GreA] [protein_id=WP_042225761.1] [location=complement(3403800..3404273)] [gbkey=CDS]
MAEEQVILTPEGLAQLEQELEELKTVKRKELAERLKTAISYGDLKENSEYHSAKDDQAFMETRIITIERMLKKARVSDNT
DTSTVKVGSIVVLNDIEFSEKVEYRIVGPAEADVADNKISYESPLGVSLMGKSVGSIISVDAPVGVIKYELLEIRAK
>lcl|NZ_CP026520.1_prot_WP_042225763.1_3071 [locus_tag=PC41400_RS15555] [protein=quinone-dependent dihydroorotate dehydrogenase] [protein_id=WP_042225763.1] [location=3404682..3405791] [gbkey=CDS]
MIYEALVKPMLFRMEPEEAHDFVLRNLQRIQRQPGLLYLIEKLYGVREEPGLASELLGIRFPSPIGLAAGLDKDARAVPA
LSSMGFGFMEVGSVTPIGQAGNDRPRLFRLPDDEALINRMGFNNGGIHAMKERLESCNALKIPLGINIGKNKRTPNEKAH
EDYTRCLRSLYPYADFFIINLSSPNTPSLRKLQYGSELDELLIRIVAETERLNARHRGDKPVFVKIAPDMDGGELENMIG
KIMRHGIPGVIAANTTVTRGGLTHPHREETGGLSGKPLHLKSTEIIRSIYKQSGGTLKIIGSGGIFTARDVYDKIRAGAS
LVEIYTSFIYKGPRINRQCSSELLDLLKQDGFASLSEAVGADHAAGSGA
>lcl|NZ_CP026520.1_prot_WP_009673902.1_3072 [locus_tag=PC41400_RS15560] [protein=L,D-transpeptidase] [protein_id=WP_009673902.1] [location=complement(3405840..3406313)] [gbkey=CDS]
MSYHIIANFSANRDNKGTLKMYDGSGALVFGPVEALGRGSNDPANNNNHANWKMTNADTPTGEYAASVIGAGTPTSSYGP
YSRVNLDPTSGNALIAENNGRSGFMIHGGDASTDSSASWYPLRPTYGCIRLSNSNQNALINKIKSVGGSGKLTVNNI
>lcl|NZ_CP026520.1_prot_WP_129112497.1_3073 [locus_tag=PC41400_RS15565] [protein=helix-turn-helix domain-containing protein] [protein_id=WP_129112497.1] [location=complement(3406520..3406774)] [gbkey=CDS]
MEKENAPLPISDEEFLHLVRLSKDGDPEAMLRLLQVFEDDILKISRYIRMSREDAVQSIVTDFIALVKQSDDVPPSCGAD
SDKT
>lcl|NZ_CP026520.1_prot_WP_042225764.1_3074 [locus_tag=PC41400_RS15570] [protein=sigma-70 family RNA polymerase sigma factor] [protein_id=WP_042225764.1] [location=complement(3406711..3407073)] [gbkey=CDS]
MSRQSTLVEKYRKEVYRIGWRIQYKARVISNREYAVSDNAYSFTAGSFTESSDSRLDMLALIQSLPSRAGRIVLHELFIQ
DKTEAQVAKDLNMTQQAVNKWKRKMLRFLSQTKNSSISSG
>lcl|NZ_CP026520.1_prot_WP_042225766.1_3075 [locus_tag=PC41400_RS15575] [protein=hypothetical protein] [protein_id=WP_042225766.1] [location=3407239..3407457] [gbkey=CDS]
MDPNIENLVSLATNVGFPVTLCFVLLRYVLQTMGEKLDKLDSSLNKLCGLIKEIDARADRGRGKEDKNGTQP
>lcl|NZ_CP026520.1_prot_WP_042225767.1_3076 [locus_tag=PC41400_RS15580] [protein=DUF3888 domain-containing protein] [protein_id=WP_042225767.1] [location=3407648..3408040] [gbkey=CDS]
MRLTLLANLANVVLAASLFVSAPIHTGASEPAHRLNNQPQSSCEVLRDAFVWSLFKPIGIVLRQYNDERQFMVDDVVEVK
RLKQGQYYWETTLKVTTFEGPHNPPYHKYIITFTNLFSNEVKVKKVLRTD
>lcl|NZ_CP026520.1_prot_WP_042225769.1_3077 [locus_tag=PC41400_RS15585] [protein=ABC transporter substrate-binding protein] [protein_id=WP_042225769.1] [location=complement(3408259..3410073)] [gbkey=CDS]
MNITEHFLRFYPLLAGSEKSEDGSFPVTIERLADLLCCSSRNVKFILRKLEDEQYIRWQPGRGRGNSSRLTLLKEPEDAV
FPLFRELMNKGKLKEAIDLLSMDALPVSLKHRLRTMLDSHLGFQVEESGGVLLDVLRVTHNKRLETHDPAYIFTTFESYL
ISQVCSRLIAYEPAEGRFVPGLAHAWQSSDDCTRWTFFLRKGVRFHHGRSLTVQDVLFTVRRLKRLPHSAWLCEDIGEIE
TDGEYTLTFKLCRSSLFFPHILSSVQFSILPADLDPASHRLTGTGPFRIAEQTERLLVLNAFDAYYGHRPLLDRVELLLL
PEDTSNERRYQISGNEKDSVTAIGTESGQTELKVNLINPVAGSRYLLFNFRKPGIHHNADFRKAMRIIFDRVAMVAELKG
NRIAPADSFLQSESEGASFEAAPLQEAVRLLRKSGYRGEPLKLFFMSKKEEEEEARWLLGRCARAGLPLILQPMKHFERS
RVAAEADLLAAEEVLEDDLEWGLISYYKNRSNYLHHLLTEEQLGLLDRILDGFAGFPEPSRMKLLREAERTLKDNDWLLF
GCHVIKKVHLHDSLQGKQMDSFGLIDISRVWIQSPSSQKAVWPV
>lcl|NZ_CP026520.1_prot_WP_042226250.1_3078 [locus_tag=PC41400_RS15590] [protein=helix-turn-helix transcriptional regulator] [protein_id=WP_042226250.1] [location=3410313..3410552] [gbkey=CDS]
MLIWMLRKIMAERGLWTGAALARLLKEKAGYDLSAPSISALLNESPKQMKADTLDALCTALDCTPGDLWVHKPSGRKGA
>lcl|NZ_CP026520.1_prot_WP_042225771.1_3079 [gene=def] [locus_tag=PC41400_RS15595] [protein=peptide deformylase] [protein_id=WP_042225771.1] [location=3410583..3411101] [gbkey=CDS]
MAVKTILPFGDPMLRKEAKPVTEVNARVLKLLDDMTDTLYAEEGRAGLAAPQVGILRRVIVMDCGEGLIELINPEIVDMS
GEQTGPEACLSYPGYAGIVKRAEYVKITTLDRSGNTIELEGEGYLARCIQHEMDHLNGILFVDRVVRNGFYQEQTGQPAD
LAEVIRLTNRKF
>lcl|NZ_CP026520.1_prot_WP_042225773.1_3080 [locus_tag=PC41400_RS15600] [protein=RNA polymerase sigma factor] [protein_id=WP_042225773.1] [location=3411268..3413052] [gbkey=CDS]
MTGQLFCEVLILNHYTTDLKNWEQMDDNSLVQQAQEGEREAFGELVRRHRSRMYGYARAITRESFLAEDIVQDALVKAFL
HLGTLVEADKFLPWVHRIVRNQAFTRLKKTSARERAFSQLAPMWKDAGEESQEWGDLDRILDRLGRTAAGKADSQLTPEE
AYMQKETLEILTGIVGCLKPRERRIFESHFFDQLSPQEIAGLFELSSANVYQIISRSRKKVMQERTRITVGSYMETRKDW
GDMKTVIVPDHESFTEVQSWTTAADALYAMLKAAGETLSHPMVMGLTGHAFRLNIVPSSVHIAGPTAYSFAGVLSKGLEN
IGYQSSYVDGMSGTLGTNANLLDPSLLGKEAMAKRGINQALPEALDLIHRSLDRGIPVLAWDLFLPEFGLIYGYDDERHL
LHAYECGKKDTLEYDNLGRSVKEEIFVLALADSAGTTFKQRLGRAMNMIVDHYDGREPDDCTGSVKGLAAYDAWIAAFRG
GKIEPNGNAYNIAVVRDARTHAVRFLNEAAEQWPLAGETAACFADVPVLAHKAAQEYKEALEHFSVLKDLFPFPQGGDPN
EPDASAKAISLLELIKEREQTAVGYLRELSDLLR
>lcl|NZ_CP026520.1_prot_WP_042225775.1_3081 [locus_tag=PC41400_RS15605] [protein=TetR/AcrR family transcriptional regulator] [protein_id=WP_042225775.1] [location=3413225..3413803] [gbkey=CDS]
MSNNQTHQQILEAAFKLFAEHGIDKTSLAMIAAEVGITKPAIYYHFSSKEALIAFIFEETFRDYTFSHYFKLDEFTKENF
QTLLVENGLKMLPQEEEEYSVMRVLNEFLLAVSRQDVYRQRLYDIEEDFLLGFTALMEKGVELGVVALDKISAKAYVLAL
VIDNLSNFMLMGFNPDVREVWAEAVAGVLIRP
>lcl|NZ_CP026520.1_prot_WP_042225776.1_3082 [locus_tag=PC41400_RS15610] [protein=hypothetical protein] [protein_id=WP_042225776.1] [location=3413869..3414576] [gbkey=CDS]
MTSFHTQPQQVTVGLDNASRIILLAGPVLLGGVLGYFIPSLARLALKLPWIPYQGPIEAITSFHGPWVSLVTALVGAAAG
LWFTGEAFKEIVFVTLSEQEIQLKKNGRVQSIPKEAVRIVFMDGKQMVILGDGGRELARETYEAPAESLPAALQPFGYTW
SGEGDPYKERYARWVPDMPGVSPPVNALLKAREQALQKKEKEEAADLRREIARLGVVVRDEGTRQYTRGTDEGRG
>lcl|NZ_CP026520.1_prot_WP_042225777.1_3083 [locus_tag=PC41400_RS15615] [protein=ABC transporter ATP-binding protein] [protein_id=WP_042225777.1] [location=complement(3414742..3415524)] [gbkey=CDS]
MDTAIIELKDVTWKTGSKELLKSVSWTVRPQEHWALLGLNGSGKTTLLNLINGYIWPTRGSVTVLGHRFGEIDLRELRQS
IGWVSSSLQDKLYGADKAQSVVISGKYATIGLYERLTDEDLNRAEVLMRTMGCLHLWDREFRTCSQGEKQKVLIARALMA
DPKLLILDEPCNGLDLFSREKLLESIRELTRQEETPTLIYVTHHTEEILPVFGHTLLLRKGEVVRQGATGELMNTATLSE
FFEAPVRVDNHDERFYVRVP
>lcl|NZ_CP026520.1_prot_WP_042225778.1_3084 [locus_tag=PC41400_RS15620] [protein=helix-turn-helix domain-containing protein] [protein_id=WP_042225778.1] [location=3415677..3416471] [gbkey=CDS]
MKLEMTTGRLELYKYEESYVNEPHRHPDWFQMTIPVRGACEFTQSGMTYRLAEGEALIQRPEEEHHFRTGPGESVLIVKL
HGSLLPVPENRGNRFDVNPAEAAEKFGVWTSEVINSRNDESLRAAESRVVHEVSRMLEAGMPPAPGHAADTYLIRTIADP
HLRDAWDYIRAHYKSAIPVETLSAIALQSRYHFIRSFKAATGLTPHQTILKLRMQEARRLLIRTSETVTAISAGLGFSSA
SQFYRSFLKDEGKTPEQYRRDNRI
>lcl|NZ_CP026520.1_prot_WP_042225780.1_3085 [locus_tag=PC41400_RS15625] [protein=chitinase] [protein_id=WP_042225780.1] [location=3416730..3417998] [gbkey=CDS]
MSQWKYPLSTKFMYRALCLALGSAILLSPLQALAASETSYAVEAGTSYSSHKDENDNAYRKGGKHKRLTQGSQLEVGESR
VLNSKEIRKEWDGIEPEYGPDKAVEAVKAALPRKDYEELFPYRLGSAQWHEAAKGKEYYKENQTDYFSYDNLIQAVTEAA
NIKYKVGTRKGSKNSQEVFRLDKEGKTETLVSRSADFNSEANRGKIIETEIVDYGTFLKEGTKKNRKRELAALLAHLSHE
TGGGWDAAPGGPLRWGLFWNENIAGRTGQNMSNFVDPDSSKLYPGFGEKRYYGRGPIMLSWNFNYGLFSSIIYGDKNILL
KNPEIVAAEGKIGFMTAILFWMTPQAPKPSAHDVMIGRWKPSKELAEKGLKPAGFGITTMVLNGLEANQPETRRAAHYRD
ITSRMGVDISGEKLDTLGMKPF
>lcl|NZ_CP026520.1_prot_WP_206735003.1_3086 [locus_tag=PC41400_RS15630] [protein=ATP-binding cassette domain-containing protein] [protein_id=WP_206735003.1] [location=3418201..3419214] [gbkey=CDS]
MNIIQVDRIVKEYKTNKRFHGRFGALKTLFSKEYERTKAVNDITFTVAKGESIGYLGPNGAGKSTMIKMMTGILVPSEGN
LLVFGLNPHKHRREISKKIGVVFGQRSQLWWDLPVVDSFELHKHIYGISDRDYLNRLEQFTSMFDMAEFLHKPVRQLSLG
QRMKAEIALALLHSPEILFLDEPTIGLDVMTKHNIRQFLRKVNVEMNVTILLTTHDLKDIEEICNRILIVNQGSCVFDGT
AAELKKLHLGQCSKKIILEFNADPGTITLPHATLFKDEGIRKHFELIPSDNGEPVSITTVLNNLSCVNDIKDISIEDVGI
DEVMRAAFVNYSQKDQK
>lcl|NZ_CP026520.1_prot_WP_042225784.1_3087 [locus_tag=PC41400_RS15635] [protein=ABC-2 family transporter protein] [protein_id=WP_042225784.1] [location=3419223..3420002] [gbkey=CDS]
MKKYFSYMQKSFFLCLSYRSEVWLRILGNVVIVYIQVSIWSSLIRHAAHYGFTLEDMIAYSIVTTAVANLLMTHVFRSVD
SKLKSGDICMDLQKPVVFPFYLFFYQLGITAFQTLFTTIPSVALAFLLFGFQPPELDKLIPFIFSVLLALLISFLFGYLM
ALISFWFLTTFALEWTFRALMLIFSGSFLPIWFFTPAWAKIAEALPFQYLGFIPSAIYMGKVAEPYLALGKGALWALLLL
LAVSFLWKKAINRLEVQGG
>lcl|NZ_CP026520.1_prot_WP_156972768.1_3088 [locus_tag=PC41400_RS15640] [protein=ABC-2 family transporter protein] [protein_id=WP_156972768.1] [location=3420072..3420803] [gbkey=CDS]
MEYRADFFLGVFGQILGYAADYVILWLLIQSFSTIDGWTWPELAFLVSLNILSYAIGASFTYTPTANMEELIRTGDFDKY
LTKPLNTLMYLIASMFNFGYIAHITISLSILLWSVTALPVQWTFLKILYLICILIGGAMIQAGALILVSCWGFLFTKSQY
LFSLYFKMKEFISYPMSIYAPFIQICLTWIIPLAFINYYPSLFLLGKQEGVLVYLTPLLGPIFLSVVIKLWFAGVNRYQG
TGS
>lcl|NZ_CP026520.1_prot_WP_042225788.1_3089 [locus_tag=PC41400_RS15645] [protein=response regulator transcription factor] [protein_id=WP_042225788.1] [location=3420980..3421681] [gbkey=CDS]
MYKIFIVEDDPKIAGLLRSYLVKYGYEVDEAEDFGRVLDAFLASGAHLILLDVNLPSYDGFYWCRQIRAHSVCPILFISA
RDSQMDQVMALENGADDYITKPYDMEIVLAKIRSHLRRAYGSYAQTQEERTVQCGSLVLYPERYVVSVDGRSAELTQKES
VLLEALMEKQGRVVGRERLLDLMWEDQHFIDDNTLNVYITRVRKKLKELGCPQAVDTVRGAGYRFGAAPPGQV
>lcl|NZ_CP026520.1_prot_WP_042225789.1_3090 [locus_tag=PC41400_RS15650] [protein=sensor histidine kinase] [protein_id=WP_042225789.1] [location=3421678..3422670] [gbkey=CDS]
MSLFWKDHRPLLLFYMAQMLLVPALYWLSGEGRPPGIVLYGMAISFAVLLLYLLFRYWQQSALYNSLDNPSSAADDSFEI
PGDSALAEAFREVLRSKERTYEQQLHAYQSKMDRHIEFMNRWVHQMKTPLSVMQLTIRELEDEQMRSLQEELDRIRKGLE
MVLYTSRLDRFSEDFRVETTNLLRVVGESVAENRRLFIRKGIAAKLHIGESLEVYTDRKWLLFMLGQILTNAVNYSAGKG
DSVFIAAFRQGNATVLEITDQGIGIAREDLKRVFNPYYTGQRGRQYHESTGMGLYLVKEIAGKLGHAVELESEAGTGTTV
RILFQPGQRT
>lcl|NZ_CP026520.1_prot_WP_042225790.1_3091 [locus_tag=PC41400_RS15655] [protein=ABC transporter ATP-binding protein] [protein_id=WP_042225790.1] [location=3422904..3423674] [gbkey=CDS]
MDILQVKDLHKTYPGKVVTHALTDIHLTIGKGEFVGIMGPSGSGKTTLLNMVSTIDRPSSGEVNINGQNPYLMKKKDLAA
FRRRQLGFVFQDFNLLDTLTVAENIVLPLTLDNRKLSEMDRKLEEVATRLGIKDILGKRVYEISGGQRQRTAIARAVITS
PVLLLADEPTGSLDSNASRVVMESLAEINEREGTTLMLVTHDPLAASYCQRIVFIKDGKLAAEIHRGDNRQTFFQNIIDT
LSFWGGNTNELSSIRV
>lcl|NZ_CP026520.1_prot_WP_042225791.1_3092 [locus_tag=PC41400_RS15660] [protein=ABC transporter permease] [protein_id=WP_042225791.1] [location=3423649..3425592] [gbkey=CDS]
MNFPRFAFNNVKRNGRAYFAYFLSSAFMVMIFFTYAVFIYHPQLDRSLMGPMTRAGMQFAAYVVFIFSFFFVLYSISMFL
KSRNKEFGILTILGADSGQINRLIFLENMLVGAVSIVTGIFSGIFLSKLFLLISMKMTGMENLPFYWPIRAILLTAGSFA
LLFLVISVLTLFFIRKNNVLELLNGSKKPKKEPKLSIWLCLLGIVLLTAGLLALRIGELEPGNIFIAAVTGIAGTYFFYS
QLTVLAIRVVKRSRKRVWRGTRLLWVSEFSYKLKDNARMLFLVTVVTSVSCMSAGFVLSFEQENRANYLNNPTAIHYTSY
DRDNSDKELKVIDGMLKDAGAAYKLLETDSFYALLKGKPGSYVDLVSVSEFNKVTDAFGKGQPGLGAINSGEAAVVYTQE
AGATNIPAGPVELEKLPGTLQVNQTRKMESLASLYSSGAALVLEDQEFKRLKSASEQNSPFRSRNFVYFVPEWNTGKLPG
LDSPESVVGNKLLAWHDELQKKRETDSFISTRAQNFLQTQQATSLFRFIGVFIALIFSISTASFLYFKLHTELAADSQMY
RALSKIGLSAGEMRKSATVQIALLFFTPIVVSAVQTVVVLGPVMNRFGVAKSLGPALIVSASFLAAQAVYFLIVRSRYVR
TLNKWMV
>lcl|NZ_CP026520.1_prot_WP_042225793.1_3093 [locus_tag=PC41400_RS15665] [protein=DUF418 domain-containing protein] [protein_id=WP_042225793.1] [location=3425597..3426601] [gbkey=CDS]
MQTDKNVRDAKRIETLDYLRGFALLGIIFINIGQMVFATTDKTAADTAVNTFMNIAVSHRFFVIFSFLFGVGFYLFVSRA
KERGDRAMLLFTRRLLLLLAIGLVHHFFQPGEALLIYSILGFLLMPFFKLKPAVVFLTGFILTIAGCFTVFLLEIWGLFL
LGLWAGQIGLFRQTDRYRRGFLITMTVSFLFLVPSYYVQEEILERTGMVDVAFAAGGLSFSVFYVTALTLILRSRLAQRL
LEPLKYLGRMALTNYLMQTVMILSLSAFFNWPERIHSSQLALTAAGILLIQMAASTWWFRYYRMGPAEYLWRLGTYGTKS
VRLSSGMSETTVKG
>lcl|NZ_CP026520.1_prot_WP_042225795.1_3094 [locus_tag=PC41400_RS15675] [protein=stalk domain-containing protein] [protein_id=WP_042225795.1] [location=complement(3427212..3428600)] [gbkey=CDS]
MFRRFVLILITITVFFFTYVKDAKADEPIRVFINGYQVAFTQLPIVENGTTLVQFRPLFESLNYKLTWNQEKQSITAING
NTNIELTIGSNNAFVNTESSELTLAPRIINGDTFVPLRFVGEVSGGDVTWEQNTNHVDIVTDKGYYLIHACLDNDLDKVK
YWLDQEANPNFYSHMDSIPLEWAIHHHNTDMAKLLLKYKADPNYEYNFVTNLLHTAIYYKNPDMVQVLIDGGADIWKKNS
EGNAALAWTNFYLEKAQNETDKNNLLKIQSKLLDLTNKQAKMWSLKASFDSRILEIVRQDFKGDIGPFSNEKSQVIYGLK
LSVMREATKETVNQLRNKLTPLGYKVFRTGDIFENRLPEQVVIIKGTDADFLKQLGEIGAIYDADPVIKQLDKWNTSTKL
EVASASSNWVVVQLLEKPKDFLSFAKEVYAFNPDYVDRGTGSVETFAKEIEKAQGFILWWPN
>lcl|NZ_CP026520.1_prot_WP_042225799.1_3095 [locus_tag=PC41400_RS15685] [protein=DinB family protein] [protein_id=WP_042225799.1] [location=3429035..3429541] [gbkey=CDS]
MSNHQAVEMYNFHNWANETIFKRLKELPESVYRQEMTSVFPTAAKLMAHIYIVDLCWIYVLEGVDMNKAGADARLNQEEI
ENMSLEELEGAYTELSRRFQDWIALGQDLDQRFMLDNPYTRKRDTSYAEVLFQVANHGTYHRGNLSAMLRQIGQSSVMTE
YALYWYNE
>lcl|NZ_CP026520.1_prot_WP_042225800.1_3096 [locus_tag=PC41400_RS15690] [protein=AAA family ATPase] [protein_id=WP_042225800.1] [location=3429667..3430428] [gbkey=CDS]
MHVESDDRFLRHIELMREKVPSWKRYPFDLDAVRSLDRLVFHPKVTYIVGENGTGKSTLMEAIATAWGFNPEGGTLNFSF
ATNPTHSELHEYIRLSKGLRKARDGFFFRAESYYNLATNIDELDADSSLPMPLIKDSYGGVSLHEQSHGESFFAAFVHRF
GGQGLYILDEPEAALSPLRQMAMLSRMHELVGRNSQFIVSTHSPILMAYPDAVMYHLSANGIEETTLEETEHYVIMKQFL
NNKENMLRELFGE
>lcl|NZ_CP026520.1_prot_WP_042225802.1_3097 [gene=rlmN] [locus_tag=PC41400_RS15695] [protein=23S rRNA (adenine(2503)-C(2))-methyltransferase RlmN] [protein_id=WP_042225802.1] [location=3430592..3431653] [gbkey=CDS]
MNKPSIYGLTFDQLTAWLQPYGHKKFRASQIWDWLYRKRVASFADMTDVNKDCLGLLEEHFVLQTMEEHVKQESSDGTVK
FLFRLSDGNLIETVLMRHKFGLSVCVTTQVGCNIGCSFCASGLLKKSRDLSGGEIVEQIMKVQFYLDKAAKEERVSHVVV
MGIGEPFDNFENLSDFLRVIKDHKGLAIGARHITVSTSGLADKIVEFADSDLHVNLAVSLHAPNNELRTRIMKINKAIPI
EKLMEAIDYYLGKTNRRLTLEYILLKGVNDAREHAEELAELIGDRRPLVNVNLIPYNPVDEHSQYQRSEKESVRAFYDIL
KKQNVSCSVRLEHGADIDAACGQLRSKQIKATG
>lcl|NZ_CP026520.1_prot_WP_255246817.1_3098 [locus_tag=PC41400_RS30600] [protein=hypothetical protein] [protein_id=WP_255246817.1] [location=3431792..3431914] [gbkey=CDS]
MLQGGDRLPDAEVNFSPLESMDRQLHVYGRSVYIIEAIAS
>lcl|NZ_CP026520.1_prot_WP_042225803.1_3099 [locus_tag=PC41400_RS15700] [protein=iron ABC transporter permease] [protein_id=WP_042225803.1] [location=3432151..3433203] [gbkey=CDS]
MIQQNAARETQAASSVPLRTRPLAAALILTAGILLLVLGIGVSVSFGAADILFSTVWEAVFHFNPESTQHQIIADLRMPR
VLGGAMVGASFAVAGAIMQGMTRNPLADSGLLGLNAGAGFMLALCFAFFPGMAYSKIILMSFLGAAGGAALIYGIGSIAK
GGLTPLRLVLAGAALSALLSALSEGIALYFHIGQDIAFWYAGGVSGTTWIQLKVMLPWFLAALLGAIALSRSITMLSLGD
EIAKGLGQRTGLVKLGGTVVVLILAGASVSVVGAVGFVGLIIPHLTRYLVGVDYRWIIPCSAVLGSLLVVVSDLAARMIH
PPYETPIGALIALIGVPFFLYLTRKERREL
>lcl|NZ_CP026520.1_prot_WP_042226251.1_3100 [locus_tag=PC41400_RS15705] [protein=iron ABC transporter permease] [protein_id=WP_042226251.1] [location=3433234..3434253] [gbkey=CDS]
MAHEKKRKRRNIAILVIFGLLLVLTFVISMNIGFIRLSPMELFKTLIGEGSRKQSLILFDFRLPRIVLSMLAGAGLAISG
AILQGISRNALADPGILGINAGAGLAVLLFICFYPAAAAAPIFLLPVLALAGAGATAVLIYTLAFKRNEGLSPTRLLLSG
IAVAAGISAATIVLTLRISPEKYQFVATWMAGSIWGTTWKFVLALLPWIVILSIYVFCKARVLDVLNLGDQLASGLGAPV
ERERFLLLASAVGLAGSCVAVSGGIGFVGLIAPHLARRLVGPGHKVLLPAAALAGALLVCAADTLGRWVLQPSEMPAGIV
VAVIGAPYFLYLLARSKTE
>lcl|NZ_CP026520.1_prot_WP_042225805.1_3101 [locus_tag=PC41400_RS15710] [protein=ABC transporter ATP-binding protein] [protein_id=WP_042225805.1] [location=3434309..3436084] [gbkey=CDS]
MLRRFFAYYKPYKGLFMLDFSCAIIAALLELAFPLAVNRVIDDLLPSGKWEWIAWGCVGLLGIYVISAALNYVVTYWGHK
LGINIESDMRTKLFNHVQKLSFRFFDNNKTGHLVSRMTNDLMDIGEIAHHGPEDLFIAVMTLAGAFGIMLGINWKLAVLT
FIIVPGMISLSLYFSRKMSLAFDKMFESIADYNARVENNVSGIRVVQAFTNENYEIAKFASNNGQFRRSKLLAYRIMAWN
SSISYLLMKIISLFVLLCGTWFVLAKQMSYGEFIAFVMLSNVFLGPIKQINSVIEMYPKGIAGFKRYLEILELEPDVADA
PDARDVDRLAGNIRYRNVSFGYENKETVLNGLNLSIRPGETVAFVGPSGAGKTTLCSLLPRFYEVTDGAITIDGIDVREM
KLESLRSHIGIVQQEVFLFDGTIRENIAYGKLGASDGEIREAAKRAQLEELILSLPEGLDTRIGERGVKLSGGQKQRLSI
ARMFLKNPPILILDEATSALDSETEAVIQEALWELSKGRTTLIIAHRLATVKHADRIVVVSKEGIAEQGSHEELLAGGGA
YSKLYAAQFGAELLNKLGSAIPREERAAGTV
>lcl|NZ_CP026520.1_prot_WP_042225807.1_3102 [locus_tag=PC41400_RS15715] [protein=ABC transporter ATP-binding protein] [protein_id=WP_042225807.1] [location=3436257..3437060] [gbkey=CDS]
MTLVQERLHTSGLHIGYGDLSIVNDLNISIPQGKITALVGANGSGKSTILKTMARLMKPASGGVLLDGVSIHSKSTREVA
KELAILPQNPSAPAGLTVTELVAYGRYPHQSGFGTVSSEDKRVISWALEVTGMMQFHARPVDQLSGGQRQRAWIAMALAQ
QTDILFLDEPTTFLDMAHQLEVLQLLHKLNSEEKRTIVMVIHDLNHASRYAQHIVAIKSGSVICEGSPADVMNQDMLREV
FSIHADIVTDPRTGVPLCLPYELAPEA
>lcl|NZ_CP026520.1_prot_WP_042225809.1_3103 [locus_tag=PC41400_RS15720] [protein=ABC transporter substrate-binding protein] [protein_id=WP_042225809.1] [location=complement(3437114..3438832)] [gbkey=CDS]
MAFSETGSLSLGSLLFQFTAADKIMPGTCEPAAVPEFDEDEKPDIHTLLLITAGTGTLRVADETFFCAPDKCYMIPPGTS
YRLRMDSDRCLTYYRITCRIVYCQGTHTALYLKPLVLGRVEWSVYPFARAVRLADLLTESEPSGELQHFYRQLQFQELMG
FLLEHNVNSPEPSSCTHAVENSIRYIQEHYRRPITVKFLAEQARVPHWQYSTIFQELTGKKPLDYLNELRIEKAKEWLRT
TREPLREIARRVGFADEYYFNRRFRQVTGITPRQYSQSVQKRTRIRDWTGHEVEIPALPRRIIYHGETFGDLAVLGVEAI
GGGYPWINQTRRLKSPVKDIGFPINPAVTEALKPDLIIVSSDDESQYDTLRGIAPTLTFDSFAPLDERIGLLGTILGRRR
EAENWLARYEEKAAAMWSRLRASLQPDETASVFTFARDRFFVMGTIGLSSMLYHPDGFRPAGRTRRLLDKKLSYLELLPE
EIKDYAGDRIFMLLPTDAGSRSAATEFMRSPLWTELRAVRSGRCYAVEADEWNFADATTTEKFLDELPRLLGKEQIVPPH
GGVQPGRKPEES
>lcl|NZ_CP026520.1_prot_WP_042225810.1_3104 [locus_tag=PC41400_RS15725] [protein=ABC transporter substrate-binding protein] [protein_id=WP_042225810.1] [location=complement(3438984..3439982)] [gbkey=CDS]
MNLARPVSTLFFTAILSAALLSGCAGSGSQKTGDTASKQTANPAGTPTAAADKSAASPRKVTDYKGHTAEIPAAAKKVIF
YGETFSDLDALGVQAVGTATTWLSGTVYADKHKNITDIGFPINLEKVLELAPDLILTGSTDEKEYEQLNKIAPTLMFNTF
LPMKERLLQLGDLLGKKPEAEKWLADYDTRSEAMWKTLRDAGIKPGETASVFTYYPGNRLFVMASTGLSQVLYDKNGFLP
TPPIQDMLDEKAGFKQISPEVLPKIAGDRIFILTPVVKEAQQSTEELLKSRIWLDLPAVKNGHVYTLDILKSGSDAATRE
WLLNEIPVLLKK
>lcl|NZ_CP026520.1_prot_WP_042225812.1_3105 [locus_tag=PC41400_RS15730] [protein=DMT family transporter] [protein_id=WP_042225812.1] [location=3440280..3441179] [gbkey=CDS]
MNPIKFGSLVVTATLLMGSSFAVGKMGLAYVSPLLLVGVRFTLAGALMALWVLGTRRRMPQTASGWGRIALIGLFQTTGV
MACIFLSLRTISAGESSLLTFVNPLLVALLGTLFLGMKYRWSQWAGVGIGFLGVAVTLGLQLDLKTGTLFGLGSAVFWAA
ATLLVKRWSASFDIWVMTAFQMLAGGLALLVLGLLLEKPLFVLNAESIGIILYLAVLGSIVQFAAWYYLLSRGDPGKTSA
FLFLAPFFGVLSGWLLLGEAVRSYVYVGGLLIFAGIFLVNWAPARTGAGSARFEEDRSA
>lcl|NZ_CP026520.1_prot_WP_241688666.1_3106 [locus_tag=PC41400_RS15735] [protein=family 16 glycosylhydrolase] [protein_id=WP_241688666.1] [location=3441305..3442537] [gbkey=CDS]
MMNRKRLSLLLSVFLLLSLFSSFVLGGTAGASSVSNWVPVWSDEFNGPNGSGPDSSKWNLETGAGGWGNNELQHYTNRTS
NAYQENGNLVIQALKENYNGSGYTSARLTTQNKFNLKYGRVEMRAKLPYGKGIWPAFWMLGKDIGSVGWPVSGEIDIMEY
VGALEPNRAYGTLHGPGYSGCCGLSGSYLHSKNLSSDYHTFALEWEPGVFKWFVDGKLFHTRTLADAGTNAWVFDKEFFF
LLNLAVGGNWPGYPDGTTVFPQKYYIDYIRVWQRAGGYNVVALKSEANGKYVTAEHAGASPLIARSQTASGWEKFELVDL
GGGKVALRALINNRYVSADNAGGSPLIANRETIGTWETFELGKTADGKQTLKALANNQYVCADNEGASPLIANRATASKW
ESFVLVKQNP
>lcl|NZ_CP026520.1_prot_WP_042225815.1_3107 [locus_tag=PC41400_RS15740] [protein=aldo/keto reductase] [protein_id=WP_042225815.1] [location=3442798..3443640] [gbkey=CDS]
MTRHLQDTTTLHNGVKMPWLGLGVFKVEEGPELVEAVKSAIKNGYRSIDTAAVYGNEEGVGQGIREGLAETGLSREDLFV
TSKVWNADQGYESTLQALEVSLRKLGLDYLDLYLIHWPVQGKYKDTWRALEAIYESGKARSIGVSNFHVHHLQDLLQDAK
IKPMVDQVEFHPCLTQEELRAYCREQGIQFEAWSPLMQGELLDNEVLRAIAGRHGKSVAQIILRWDLQNGVVTIPKSTKE
HRIIENASVFDFELTADDMSRISALNENRRVGPDPDNFDF
>lcl|NZ_CP026520.1_prot_WP_042225817.1_3108 [locus_tag=PC41400_RS15745] [protein=HAMP domain-containing sensor histidine kinase] [protein_id=WP_042225817.1] [location=3443855..3445363] [gbkey=CDS]
MSIRMRLTLLYSGILSATLLLFGIVLYSFLQIYFINDLKDSLREQTDVLQESVTYKLKVDPLGSDLDIQLDHVDTVQSGM
YLQLVNLINGDKARTRNLGQVELPFSKDAAGKNKKGYYITEEVQQSPFLIYNQPLLLNGKLVGILQSAYNIGLIDKFLSI
LRLVLILLSLVVVCTASFMGWLSSKRALKPVYALIEATRQIQKSDDLGNRIAVKPTGDEVSLLGTTINGMLERIQHMYGE
LDKSYMTQRRFVADASHELRTPLTTISGNAEFLQKAWLSFRQSPEQRINKDQLELSIEAIHDIADEAGRMGRLVHNLLTL
ARADSGLHLRKDRLNISSVVQDVIRKSQVLVKSAEFQAGPLDGLESLHVVGDRDYLQQLLFIFVENAFKYTENGCVRFFA
RAAGDNVELIIEDTGIGMDETDIPHIFDRFYRADPSRGETPGTGLGLSIAKWILEEHRGTVEIRSRRNEGTRFTIRLPVV
SALREKGRMPALSEAASAGIDG
>lcl|NZ_CP026520.1_prot_WP_042225819.1_3109 [locus_tag=PC41400_RS15750] [protein=oxalate decarboxylase family bicupin] [protein_id=WP_042225819.1] [location=complement(3445512..3446672)] [gbkey=CDS]
MSKPVKEGFQIPQPIRADGAGATDPGPRDVLRDLENPNMLVPPATDAGLIPNMKFSFSDAHMQLNHGGWSREVTVRELPV
AKTLAGVNMRLTPGGVRELHWHQQAEWAYMLTGKARITAVDEKGRNFIDDVGEGDLWYFPPGIPHSIQGLEEGCEFLLVF
DDGNFSDLNTLSISDWFAHTPPDVLAVNFGVDESAFENIPSGQVYIFQDKVPGSLTSLEIESPYGSVPRSFKHRLLAQTP
EVMPGGSVRIVDSSNFPVSVTVAAALVELKPGGMRELHWHPNQDEWQYYLSGQARMTVFGGNGSARTFDYRAGDVGYVPF
AYGHYIQNTGEETVWFLEMFKSDRFADVSLNQWMALTPRELVRDNLHAGPQLLDSLHKRKRPVVSS
>lcl|NZ_CP026520.1_prot_WP_042226253.1_3110 [locus_tag=PC41400_RS15755] [protein=AraC family transcriptional regulator] [protein_id=WP_042226253.1] [location=3446900..3447892] [gbkey=CDS]
MPSSIQINTMHDYFDRFSEHMGGQITGSGGERTLLLSPRLGEGTLTRIPIRTGMEIVISDFELIRDLKVQVHGAYSLFEL
NYCLEGDIYCAWEGNELHTGSLSGNVFFMENMNVYMEKKGGVRNRTVEIRLHPELLLRCAEGPEETEQIRQLLSRNRGRI
GRFTDTPLIRQGVRDLLQSGHRGSMRRWYAESKAMELIALISQPDDGVKEASGISLTPDDRNRLSEARRQVLDRLDSPPS
IPELARLCGVNEYKLKKGFRTLFGTTVYELVRRERMKRAAEYMDRGMNVSEAAVQLGYANMSNFTAAFRKQFGLNPGQYL
KQANQERIRP
>lcl|NZ_CP026520.1_prot_WP_042225820.1_3111 [locus_tag=PC41400_RS15760] [protein=MFS transporter] [protein_id=WP_042225820.1] [location=3448250..3449455] [gbkey=CDS]
MKKRFLLIFLTVLAAMTSIATFNPILGLLARSMGLSEIQTGSLVSVTGLFWITGTFLWAKWARASRKPVMIAALLGYIAT
LAAFAVLSDLARSGEPDPTRLYIQLLFLRAAGGFFFGAIPAMAQSYLIEWTTAENRASGMALFGAANGLGFVLGPALGAL
LTPVGLTAPMYASAVLLTLTAAAFWLMIPGGQPAVAGTRAGKLSAADPRISLYLAIGLILSIVMIILQVTCGIFIQDRMS
VTSQDAARYIGLGLSSAGVLVVVIQLLIGRYLKWPTGRLLKIGLSSLCAAFLLFLLWPAMYLLVFLLFGIGIGFTLPGYI
TAASLAVTEEEQPSVASFTAAVQGVGSFAGPLTGTLLYSLHTSVPYAVCALLTAAFVLAVQARTRRGAVRQAVSAPPDRT
R
>lcl|NZ_CP026520.1_prot_WP_042225823.1_3112 [locus_tag=PC41400_RS15765] [protein=GNAT family N-acetyltransferase] [protein_id=WP_042225823.1] [location=3449593..3450141] [gbkey=CDS]
MPNMILLNDQISPEIKASIAGTEQYPQIERLMVETAKWLRSKGSTQWSGLLSGSDDHDMAGALERKEVVVFKAAEGTLLA
CAILQQEAEDWDRKLWRLDPGETGTSIYLHRLTVNRENAGRNLGKTVMDWIRQEMSFPGKDRIRLDCIGSNETLNAFYKT
CGYTYIGESDGFSLYENMLPVR
>lcl|NZ_CP026520.1_prot_WP_042225825.1_3113 [locus_tag=PC41400_RS15770] [protein=DJ-1/PfpI family protein] [protein_id=WP_042225825.1] [location=3450535..3451884] [gbkey=CDS]
MKVVLRITIYLIVFVVVVGGAGAIGFINTMNNGMSIYDKAVPALTNVQVPKYDAGKPTVAVLLANEVTEVFDFLVPYEMF
AMTEAYNVYGVAPDREIKSLTGGLDVVPHYSFDEMDAMLGKSPDIIVIPFMPILDDNKYVPVREWIRKHSDVNTTLLSIC
NGAENLADTGLLNGKSAATHWGDINRLIKKYPDIQWVKDQRYVPQGNIVSSAGLTSGIDATLYVISQQLGDAVARKVAEN
MNYPSYDYVTNPKMKPFNAGLSDITYVLNNAYQWNKVKAGVLLYNGADELDLSAAFDTYAASGTTTTLTVSSKNEAIVTK
HGLNLVARYQTKDVPKLEKTIVVGADAESEAAEDIKQWNNSGNAKLMFLHRDMSGRFAMDPAFEDLAKQEDIQTAKFAAK
RLEYRATDHLKLEGNPFSFEAFGIPILLGILSLLVARFIDRRLIGRKKG
>lcl|NZ_CP026520.1_prot_PC41400_RS15775_3114 [locus_tag=PC41400_RS15775] [protein=TIGR00730 family Rossman fold protein] [pseudo=true] [location=complement(3452231..3452739)] [gbkey=CDS]
MNRCILWLGFSEGASSIFKESDLN*VRSLPDARSPLFT*EPMWG*WELLRMLCSNKAVK*SGYSPIFCKIGKLRIRARQS
SLWLIPCTAVKQRWPNWQTDLLLFLVDPE*WKNISRSLHEVSWDYIKSHVVC*ISINILIH*CPCLKLWNVNNSCSRNIP
SGFLKLILV
>lcl|NZ_CP026520.1_prot_PC41400_RS30335_3115 [locus_tag=PC41400_RS30335] [protein=transposase family protein] [pseudo=true] [partial=3'] [location=3453027..>3453197] [gbkey=CDS]
MDILNLSSFTVLSISENEHDFLIRVETNSPPSHCPHCGCEANLYKHSNREQLCMDLP
>lcl|NZ_CP026520.1_prot_WP_084706495.1_3116 [locus_tag=PC41400_RS15785] [protein=FAD dependent oxidoreductase] [protein_id=WP_084706495.1] [location=3453774..3455264] [gbkey=CDS]
MMNTYDIGNREKRRALVMGGSIGGLLTARVLADFYEEVLVIDKDEFPGEPADRSGTPQGYHPHRFTPRGKTIVERFFPGY
EEDLFKLGALSSLNKTVTNMNEYGTISGPYQRNDLKFSRALLEWVIRERVKRIPQVSFLPGHEAVQLMSTPDCKKVTGAV
IRGRSSGGEETPCEADLVVDTGGRLSKLTKWLEFMGLEAPASDYLNVELGYSTRRYRVPAAKEHLVKEWDVINIAGQPAA
GTFTGVLSFIENNVAEMLLYRPGGQYPPTKEDEYEKAAANLPDPVIGEILGELEPAAPPRGFRVPQLYRRHFEQMTDWPA
GLLVLGDAYCIYDPIFGQGMTVAAIEAETLEACLRKQSLAPVPSPAFERSVLRGLQDVIEPAWWLNCAADLRWEGVTYSG
AAPLKGIEFGRKYMSLVLKYATAEKNMKFFGLYWAVNTLTLPPGQLFHPDIVREVLSSSEEGRRLLDELLAEGSGSLDAV
LVEIVPAFSETAYETV
>lcl|NZ_CP026520.1_prot_WP_042225827.1_3117 [locus_tag=PC41400_RS15790] [protein=TetR/AcrR family transcriptional regulator] [protein_id=WP_042225827.1] [location=3455311..3455907] [gbkey=CDS]
MAPLNDEQLSQIRDERRIQIMGAALRVFARRGIIGTKMSMIAAEAGISHGLLYHYFTSKEELFTTLVQGAMEASQEAMKE
VYRFPGTPLDRIRRLSEEMLDKSGALYFLLIHQARTSEGVPDKVNQLMRDYSLDTYVEELLPLFREGQERGELDAGDTEE
MIASYISVLSGLMVVTADGNDDYRIPPVDRLLRMITKR
>lcl|NZ_CP026520.1_prot_WP_241688667.1_3118 [locus_tag=PC41400_RS15795] [protein=chitosanase] [protein_id=WP_241688667.1] [location=3456206..3457159] [gbkey=CDS]
MNLQTKQPVSVRKRVKSGMKAVLSVGLASALLLTSVLLPGNALANSNPAEGEAYAAPVSSHDSNFSPATLQFLKSKTGLD
GEQWDNIMKLVNKAEQDSLKWTEYYGYCEDIDDDRGYTIGIFGATTGGPNDTGPDGPALFKEYDAVKGAANPSVQGALAR
IGVNGTMKGAILKINDSEKVFCGKIKSLQNDPAWREAMWHTFYNVYIKYSVEQASKRGFNSALTIGSFVDTALNQGAEGD
SGSLEGILAKSGNSTNEKTFMTNFYAKRTLVVDTNEYNQPPNGKNRVKQWSSLLNMGETDLKDADAAVKKVTGWTMK
>lcl|NZ_CP026520.1_prot_WP_042225828.1_3119 [locus_tag=PC41400_RS15800] [protein=DMT family transporter] [protein_id=WP_042225828.1] [location=complement(3457366..3458226)] [gbkey=CDS]
MILFALLLVMLSGLTHSIWSLFAKSSRNKSVFLWSIAMIAVVVLLPPSIRELRTASLDSGAWLLLLASAVLQAVYGLLLS
RTYQMGDLSQVYPIMRGTNTLLIPAFGVVFLGESLSVVGWIGLLCMILGFVMLSGLPTRKSAARGEGFSPAPVLMALGVG
LCTTSYVLVDKINLQHVSPLTLLEVSNIGFAAGLTPAVLSSRQLSSEWKLNGKRIVLGAVLNTGSYLLFLLALAYAPVAH
VSPVREIGTVFATLLGIFVLKEKQGTLRILSSLVITAGIMLIGFFG
>lcl|NZ_CP026520.1_prot_WP_042225829.1_3120 [locus_tag=PC41400_RS15805] [protein=DeoR/GlpR family DNA-binding transcription regulator] [protein_id=WP_042225829.1] [location=3458557..3459327] [gbkey=CDS]
MSLTYEDRRSTILKQLDMEGKVQVHLLSAEFNVSTETVRRDLDRLEKEGKLRKVYGGAVKVRSDRVEPPFLKRAQMMQRE
KASIGKLAASLIGEGETVILDNGTTTIEILGHLRNRPDVTLITHSVPILAAAMDTFAGKIIFAGGEINRHYQAATGALTD
QLLDQFKVNKAFISVGGISLVDGITDYHLDEAVLSRKIMQRAEESILVADHTKFGVTTFARVSKLEEVSMVITDPGCPRD
WIDSVEALGIEMRISS
>lcl|NZ_CP026520.1_prot_WP_042225830.1_3121 [locus_tag=PC41400_RS15810] [protein=MFS transporter] [protein_id=WP_042225830.1] [location=complement(3459614..3460888)] [gbkey=CDS]
MQKDTLLGLFKNRNYAHLFGAQLLSQIGSVTGLTAFAFYMLDKFSTQPAYATLTEMMYSLPTLFLFFLTGVATDRFDRQK
IAVSCDAVNAGLSLLLLGAIAWGWMPLIFAVLFIRSAVGKMFHPANAALVRGVLDPAQMPAAMGMNQMLMSAFVLLGTGL
GAVSYWHSGILGSVAIDMVSFVISAVLIMRCNIPQEVRQPNGPASWRQLRLKQVGQDFVSGLRYVRHHSIVLSLLAGILI
LGLANGGGSVMYLFSLKYKLAPDTYESLQVGMTAVLGAGMLLGSIVSVRLARKIPLYGMIIASFFLGAFTHLLQAMASDT
VTFMALYLLYALSIPLCNVAFFGWLAQVTEARMMGRVQALIQPIMMLTFTAMQAWIAYAFPAHMSVESIFYVVGAAELAL
GVYYLLVLPPLARKRAEQTAKQAA
>lcl|NZ_CP026520.1_prot_WP_042225831.1_3122 [locus_tag=PC41400_RS15815] [protein=winged helix-turn-helix domain-containing protein] [protein_id=WP_042225831.1] [location=3461084..3461983] [gbkey=CDS]
MAYQIETAFSPVHECLLSFLLYKRRQNIKYLYVGPSWLQQIRDMMEPAYLDKLDALDDLTFVDLLSVLVDQCPDAASMET
FLEWVNTLSAGELYERLASYMADGKNPVWLHLGEQRDLFTELLAVWQEKYMARWTGMDALREGTREVEEAVRTGKKAEDI
VETFVTGMRVELTAIRKVHLIPAYHFRPLHTFAVLKDKINILYPHNIEKPDIFTVSAAAKALSDEKRLEILKLLSVERRS
FTDITKEIGGAKGNVHHHLMTLRMAGLVSVHLTDEATFLALRPGLSERLKQQVDHVLKG
>lcl|NZ_CP026520.1_prot_WP_042225833.1_3123 [locus_tag=PC41400_RS15820] [protein=APC family permease] [protein_id=WP_042225833.1] [location=3462338..3464155] [gbkey=CDS]
MVGKFKRLLIGRPMKSAEIEGEKLNKLKALAVLSSDALSSVAYGTEQILIVLMAAGFTALWYSVPISLGVLGLLLILILS
YRQTIFAYPTGGGAYIVAKNNLGVHTGLLAGGSLLVDYILTVAVSSSAGTDAITSAFPSLHDHRVLISLTMILFLTIMNL
RGITESASVLALPVYLFVVSIFVLIISGVVKYFTGGVHAAVPEMSTTVSNISLFLILKAFSSGCSALTGVEAVSNAIPNF
KKPAERNAAATLVMMGTILGAMFIGISLLAYWYGIAPSEKETVVSQIANATFGRGFIYYFIQGVTALILFLAANTAYSAF
PLLAFMLAKDKFMPRMFMVRGDRLGFSNGIINLGVLSALLVIVFNGDTENLIPLYALGVFIPFTLSQLGMMVRWIKLKPS
GWLGRFIVNTIGMLTTLTISLIFVFTKFSQIWMVFFFLPFVIFIFIQTKRHYENTADQLRIDLARDKPLIKGNTIVVPIA
GITQVVKQSLGYAKSVTDNVVAVYVGFNDEEIERMEKKWEEWDPGVRLIVLRSSYRSIIRPLIKFVDTVEWKTAETDHIT
VLIPQFITKHWWQHILHNQTSLLLRAYLFNRKDVVIATVPYHLNK
>lcl|NZ_CP026520.1_prot_WP_042225834.1_3124 [locus_tag=PC41400_RS15825] [protein=biliverdin-producing heme oxygenase] [protein_id=WP_042225834.1] [location=3464419..3465030] [gbkey=CDS]
MTTTIVSRLREETAPFHDQIEQNPYAKAIMDKSLTLETYKTYLEKFYGFIVPSEKALSALPEWEERGFDIESRLKTPLLE
NDLEQLGLTKRQIEELPRCGNLPDVSTLPRALGYLYVLEGSTLGGQLITKQLKAILSVDPEVNGRYFNSYGAELRTKWSE
FREFLLENVKEEEQEQVLTAAKETFILLDQWLKEDSGSHTAVI
>lcl|NZ_CP026520.1_prot_WP_042225836.1_3125 [locus_tag=PC41400_RS15830] [protein=diguanylate cyclase] [protein_id=WP_042225836.1] [location=3465053..3467608] [gbkey=CDS]
MTDRSKAKDPEVLNRTLLTEGNFASPNEEIDLTNCDKEPIHIPGLIQPHGVLLAVTNTTEQPLIVQCSRNVDLMLGRTTE
ELLGRPLEDVVGKAQLAAMQEELGSRTFDPSKLHYMNISIEVGGDPVSFYGIMHESEDLLILELEPASDKEEPVFQDYEW
IQTFFCKMKRTGNRVEASQMAAEQIKEMLGYDRVMVYEFDEEWNGKVIAEAKEEGLEPFLGHHYPASDIPKQARELYLRN
WLRMIVDVYYKPVEIVPTLHPATGRPLNLSLSVLRSVSPLHIEYLQNMGVGATTTISLIHDNKLWGLVTCHHYTPKYITH
RVRNLCNFLGFFFSSELYQRQQLDDYQTEIELKTMANRISNIFIGNTSANRVMEQLKLEEKSLLQVMNASGVAVCYKDKL
AIFGSAPEKEQIRELAFWMSRQADDYEYHTSRLSQEHPPAAAYKKTASGALYLALSPDGKDYVIWFRPEVVQVVNWAGDP
SKAVLRENDGVRLSPRKSFEKWRQVVESSSLPWKIQELRQLSDLKSIMLKQTENQMRQAEEQAQQNSRILKENEKRYLQL
MELSPVAFFTITDNRVVYCNEQGAELFEAANPEELIGKAVANFVHPEFLPVLQRKMAELDANALSLFSMSGRYVTVAGGD
RQLEITMASVVYDGKPSLFAIARRREEGEKEVFSEVSNQLQTFINTDSLTEMPNRRFFDENLEKEWKRAAGAGSPVSLIL
MDIDHFKAYNALYGYQGGDTCLQWMSDILVAIGKPNEAVIARYGGGTFVLRLEAAIEKAVDLAEQIRLGVLSMKIPQTHS
DSGDYLTVSLGVSSAYPEPGSGPDDLVYAAERALQKAKNEGRNRVSVIRPV
>lcl|NZ_CP026520.1_prot_WP_156972770.1_3126 [locus_tag=PC41400_RS29845] [protein=hypothetical protein] [protein_id=WP_156972770.1] [location=3468563..3468736] [gbkey=CDS]
MNKASVIKMLDSDGELIARLSIKVKAGNILRIRNQLYRIDALRSKERICYAYSIHKD
>lcl|NZ_CP026520.1_prot_WP_042225837.1_3127 [locus_tag=PC41400_RS15835] [protein=hypothetical protein] [protein_id=WP_042225837.1] [location=3468829..3469200] [gbkey=CDS]
MSVMYMEELFPEATQADIARAKVALRKYRENKQKVLLFESEPPETEIQIRRQAALIKSTRRIEKAISQITFADVRSVMEY
RFIKGNSRAAILYFSGWHCCEKTIDRKITEGILSVANTLLYFD
>lcl|NZ_CP026520.1_prot_WP_053228601.1_3128 [locus_tag=PC41400_RS15840] [protein=PIN domain-containing protein] [protein_id=WP_053228601.1] [location=3469582..3470823] [gbkey=CDS]
MKDTFSGFYHPSDEEFKKLWSECVFIFDTNSLLNIYRYPKDARELLLKILKNLSDRIWIPHQIAYEYHRNIHGEFINQNN
AYQETKDIITKAIGTLQQELKKLRHSNIDVSQVINKIGNIAEESKLLLDLQKELQPNLDLVKEEISSIIKTNVGEEYPQE
MLDEIYKEGLERYAKHIPPGYMDSADKKDKISYYNGKIYKDQYGDLVYWKQLLEYAKKEEIKSVILVTDDRKDDWWYKVR
GQIKGPAPELILEFNKEVQKQFYMYQTEQFIKFAKVFLAIDADTNMEEAIKDIQEYKNTGGYVTHHNPIENYFAYLIILH
KLDEVGNEDIERNISIGIKNFTGNYPVTCIVHQEEDVLVHVLIETVDPIPYDKYFHFVHESITNAIIPNPEETNFGISSM
QYLKQHEMSNYIS
>lcl|NZ_CP026520.1_prot_WP_042225839.1_3129 [locus_tag=PC41400_RS15850] [protein=Nramp family divalent metal transporter] [protein_id=WP_042225839.1] [location=3471738..3473015] [gbkey=CDS]
MAEQGITKYKRIPLNAESVLNGDVKGLKRLLPFLGPAFIAAVAYLDPGNFATNITAGSKYGYLLLWVIAVSNLMAVLIQA
LSAKLGIATGMNLPEVARERFPKPVSIFLWIQSELVIIATDLAEFIGAALGLYLLFDIPMLPAALITAVGSFAILELQRR
GYRSLEAGIAGMILIVVCSFALQVINAQPDVGAVATGIITPKFQGIDSILLAAGMLGATVMPHAIYLHSSLTQKRIVGKN
EQEKKRIFKLEFIDIVLAMVIAGAVNMAMVIISAALFFKNGLGVEELDEAFHKFYELAGPVSAVSFGLALLIAGLSSASV
GTLAGDVVMQGFINKRINLYLRRAITTIPPLVIIIMGIDATRALIMSQVVLSFGIAFALVPLLLFTSNRSIMGGLVNQRW
TTVLGWIIAAIVILLNLFLVVTTFI
>lcl|NZ_CP026520.1_prot_WP_042225840.1_3130 [gene=hflX] [locus_tag=PC41400_RS15855] [protein=GTPase HflX] [protein_id=WP_042225840.1] [location=complement(3473192..3474484)] [gbkey=CDS]
MERHKQKAILIGIHPGNREDFGYFMEELANLAAAAGIEEAGMLTQKLARSNPSHYIGKGKIEELLELIEAHGADLVICND
ELTPSQLRNLEAALDRTVMDRTALILAIFAERARTREAQLQVEVARLHYMLPRLVGLRESLGRQGGGAGLKNRGAGETKL
ELDRRRIEERIVMLQKELEQLVSHRQTQRKKRRKTEVPVVCLVGYTNAGKSSLMNAMMEAHNPGPGKQVMAEDRLFATLE
TSVRSIQLQDRKSFLLTDTVGFVSQLPHHLVKAFRSTLEEVAEADLLLHVVDSSSPVYEQQMAVTEETLKELGADHIPKL
IVYNKADLSSLEFPSADNGHVVISAKLRAGISELTEQIRERIFRHEVHCELLVPYDQSHVAAYFNEFARVRSVRYEADGA
RLTVECPESVLERYRGSYIPLSDGQAYPDN
>lcl|NZ_CP026520.1_prot_WP_042225842.1_3131 [locus_tag=PC41400_RS15860] [protein=acyl-CoA dehydrogenase] [protein_id=WP_042225842.1] [location=3474954..3476006] [gbkey=CDS]
MIFTQEEVQRIREQSPGMESEGALTEELLNLIYQNRWFKLFVPDDLNGRMSSLPDALRIFRECSRVDGGLGWLVTIGAGG
GFFASLMKPEVGAALFADEKAVVAGSGAPTGTAKRVENGYMVSGRWKYCSGAAHATMFTANCIVEAPEGEEPGENAEIRA
FIFTPDQVRIHPDWNTFGLKATGSHSISVADLFVPDRMTFLFTEVHAYGDELIYRYPFIPFAQASFAAVALGIAAHFAEA
ARDYIRVKERKANVQRKLEEQEALLSRTESAFYSAITRSWDELVLHGSLSEAGEKQVSEQCLSAVRSVLTGTFELFSLLG
LSAAMEDSPVNRTWRDLQTVCQHSLLLPEE
>lcl|NZ_CP026520.1_prot_WP_042225844.1_3132 [locus_tag=PC41400_RS15865] [protein=TetR/AcrR family transcriptional regulator] [protein_id=WP_042225844.1] [location=3476184..3476705] [gbkey=CDS]
MVKKNTSSASRRTDIVSAAIEVFAEIGYYRATTAQVAERAAISQPYVYRFFTKESLLVEALEVSWQRIETAFRQVLESAS
PDTLERGFIEAYEGILAAHGSEILLQMQAQTIREEAVRKTMQEGMSRVKRLVLNAFEEAEMEDAERRTSDFLARGMLCNV
SMAIGLPDLMLKK
>lcl|NZ_CP026520.1_prot_WP_042225845.1_3133 [locus_tag=PC41400_RS15870] [protein=SDR family NAD(P)-dependent oxidoreductase] [protein_id=WP_042225845.1] [location=3476777..3477739] [gbkey=CDS]
MKKAVVLGATGGVGTTLTEELVKRGIETVAFGRSFAKLEKLKLELGNPPHLKLVKGDAFDSAGIIKAAEGADVLFHSANI
PYHEMESRLLPLGEAVMKAAEHLAVKVVVVDGIYPYGRRTGAAKAEEDHPKQPHTRKGKVRFAYEQLIFASRWKRAKPLI
VRLPDYYGPSAQASYLSVTMEAIAAGKPAVFIGNMTVPREYVYLPDASRMIVEIAGRDGTEGQNWHIPGPGLLSGKELVQ
LAQRASGKRKAVYPLGRVSLSLAGLFNPVMKEVVEMLYLTEEPFVLSGRKYEKEIGPIIWTPHEKAVSDTIHALMNRNKA
>lcl|NZ_CP026520.1_prot_WP_042225847.1_3134 [locus_tag=PC41400_RS15875] [protein=DUF3892 domain-containing protein] [protein_id=WP_042225847.1] [location=3477826..3478152] [gbkey=CDS]
MNEGSTNTDFEAIYENYKQQGELQAVQEQTEAAADGLESIVAVRKNEDGDIIAVKTSSGRELDYVSALNEAKSGNLAHVD
VFHRYGRDILRSEPDGIKSNNLDQLPHF
>lcl|NZ_CP026520.1_prot_WP_053228602.1_3135 [locus_tag=PC41400_RS15880] [protein=hypothetical protein] [protein_id=WP_053228602.1] [location=complement(3478206..3479570)] [gbkey=CDS]
MLKKWFLSLAIAGSALLSGLAGQAEASASVLPAAAPPVVKGGELTSPTDFKSQRDALPESRKPLEAPLSLDKAARGQGLS
VNVNAALNNNPNNAIQVVTGAVYADKITEEGGQRWYVFQTNAPGKLTAYLQTIPSASIDYDLHLFRYDPATSSLVEQETS
SYGPQSNEQISRIAPAGIYFLAVSAYTGFDANTSYYFTVVESPSYDSAEPDDNIWHAQEKTDSFTVSGQTIDNSLDADWI
KYTVTAAKSFHIKLANNAPGTTYQLQIFNQNLGSLGTVDQNKDVDASLTPGTYYLRVVSLNSYNASVPYSLTFSEYKAPA
SVEIVSIKSEPNVDAGKRIDYGEGNKWRIQNYMTVTGLAKDASGAPAFNAPITVYIQTKLNNKIYSGTGTTSSNGTFNVA
INGIQPAIGQNLFYGYVSTHYYDIIPLLVYSGGTQLNANDNSLYHYAYSIYNPH
>lcl|NZ_CP026520.1_prot_WP_042225850.1_3136 [locus_tag=PC41400_RS15885] [protein=ABC-F family ATP-binding cassette domain-containing protein] [protein_id=WP_042225850.1] [location=3479933..3481489] [gbkey=CDS]
MSILNVERLSHGFGDRAIFNDVSFRLLKGEHIGLIGANGEGKSTFMNIITGKLQPDDGKVEWSKRMRVGYLDQHAVLSKG
MTIRDVLKGAFQYLFDMEQEMNGMYEKMGDVTPEELEQLLEDVGTIQDTLTNQDFYMIDAKIEETARGLGLTDIGLDKDV
NDLSGGQRTKVLLAKLLLEKPDILLLDEPTNYLDEQHIEWLKRYLQEYENAFILISHDIPFLNSVINLIYHMENQELNRY
VGDYDHFKQVYEMKKQQLESAYKRQQQEIADLKDFVARNKASVATRNMAMSRQKKLDKMDVIELAKEKPKPQFNFKEGRT
SGKLIFETKDLVIGYEEALSRPLDLRMERGQKIALVGANGIGKTTLLRSILGEIPALSGSVQLGDNLSIGYFEQEMKEAN
YNTCIEEIWKEFPSYTQFEVRAALAKCGLTTKHIESKITVLSGGEKAKVRLCKLINNETNLLVFDEPTNHLDVDAKEELA
RALKAYKGSILLISHEPEFYREVVTETWNCESWTTKVF
>lcl|NZ_CP026520.1_prot_WP_042225852.1_3137 [locus_tag=PC41400_RS15890] [protein=helix-turn-helix transcriptional regulator] [protein_id=WP_042225852.1] [location=3481666..3482460] [gbkey=CDS]
MTDRYSEIDEVILYIQRNIYEPLSLNQLASYAAYSPYHFTRLFKEKIGLTPQYYVSSLRLQKAKDLLLRTNLSIRDIGLE
IGQQSLGTFTTRFTEKVGMTPSEFRNSTFLADSHFHSLKKLDDWSMPETAALPFGNIEGIVQAAIPFEGVVLIGLFAKPI
PEGLPLYGTLLSSLGSFRIPGVKPGTYYLMATSVSWGTKAVDFLLPEATLRTRSREPIVVEPGIPVPQQLVTLHVPRYDD
PPILISLPVLMNNFLTRTTPPGSR
>lcl|NZ_CP026520.1_prot_WP_042225853.1_3138 [locus_tag=PC41400_RS15895] [protein=VOC family protein] [protein_id=WP_042225853.1] [location=3482571..3482990] [gbkey=CDS]
MAKHTTYIFSEDARAQAEFYVQALGGEITSVTTYGEVPDANEEELKDKVINLSLVAAGVTFLMSDSMFESLNPGNVVSLC
LEFPTEEEAHTAFDKLSQGGTVKQPLEPAFWGALFGQFEDKYGISWMITSDTSACRQSQ
>lcl|NZ_CP026520.1_prot_WP_042225854.1_3139 [locus_tag=PC41400_RS15900] [protein=radical SAM protein] [protein_id=WP_042225854.1] [location=complement(3483145..3483963)] [gbkey=CDS]
MKPEYGYTMPKTLLNKGTGFLAGYTHSLNPYTGCSFGCSYCYVRKMPVSTFRGAEWGTWVDIKKEAAELFRKELRRAKSK
GPVTLFMSSSTDPYQPIEYKERVTRSLLEVMTEEPPDFVLLQTRSPLVERDMDLLLQLKNQVRVSMTVETDLDEIRKRFS
PAAPPIQARLKTLKALKAAGIPAQAAIAPLLPSSEAFPELLKSVVNRVCVDDYFMGDGSGGRRTRSLNMETAYKELGMED
WYHPHAYRFMVERLKKVFAENQIFISQKGFEP
>lcl|NZ_CP026520.1_prot_WP_042225855.1_3140 [locus_tag=PC41400_RS15905] [protein=bifunctional transcriptional activator/DNA repair enzyme AdaA] [protein_id=WP_042225855.1] [location=complement(3484036..3484635)] [gbkey=CDS]
MSERFRKAAKLDSPSADKWQAIISNDSSYDGVFFYGVKTTGIFCRPSCKSKAPKPENTCIFQDAEQALQAQFRPCKRCKP
TGRRLPEDDWIEQIAHYADNHYKQPLTLRTFSEICHGSPYHLQRTFKRIKGMTPLEYLRHIRIENAKSLLGNSELSVSDI
GSKVGIPNTSYFITLFKKKTGYTPAGYRQSLSLTRTIEN
>lcl|NZ_CP026520.1_prot_WP_042225857.1_3141 [locus_tag=PC41400_RS15910] [protein=VOC family protein] [protein_id=WP_042225857.1] [location=3484874..3485290] [gbkey=CDS]
MATKADKIFVNLPVKNLDKSVQFFTALGFEFNDQFTGENATCMVVSDSIYVMLLVEDFFKTFTNKEIADTEKSAEVIIAL
SADSKEQVDEIVNNALGAGGSPLNDKVDHGFMYTWSFRDIDGHHWEFIYMDESAAAQG
>lcl|NZ_CP026520.1_prot_WP_042225858.1_3142 [locus_tag=PC41400_RS15915] [protein=neutral zinc metallopeptidase] [protein_id=WP_042225858.1] [location=3485497..3486333] [gbkey=CDS]
MKWQGRRGSSNVEDRRGVGGKTLVGGGIGGIIIVLLMTLLGGNPGDLLSDLTGAGSGSNAPYEETQQEKELSQFVSVVLA
DTEDVWTAQFKERGLVYEKPTLVLYTDQVQSACGTAGSSTGPFYCPGDRKLYIDLSFYNELKQEFKAPGDFAMAYVIAHE
VGHHVQTLLGTSDKIMPLRQKMSEAEFNKYLVRFELQADYFAGVWAHHAKGMDVLEEGDLEEALTAASAVGDDAIQKRAR
GYVVPESFTHGTSEQRKSWFSKGYQSGRIEDGDTFSSK
>lcl|NZ_CP026520.1_prot_WP_042225859.1_3143 [gene=pdaA] [locus_tag=PC41400_RS15920] [protein=delta-lactam-biosynthetic de-N-acetylase] [protein_id=WP_042225859.1] [location=3486499..3487284] [gbkey=CDS]
MKKVTIVFALLTIFITVFPLDTLAADGGNKPYHFGFKKSKQGKPASIDEEGFKSILEKHDAVFLGDTGKKELYLTFDNGY
ENGCTPKILDVLKAKQVPAAFFLTGHYVEDKPELVKRMTAEGHLIGNHSWSHPDMSQISDEKIKTELDRVKNAVVNLTGV
QTMSFMRPPRGIFSDRVLEVSKQHGYINVFWSIAYKDWDVRDQKGWKYAYDKVIAQLHPGAVILLHSISTDNTDALDKII
DEARKQGYEFKSLESVNKKTE
>lcl|NZ_CP026520.1_prot_WP_042225860.1_3144 [locus_tag=PC41400_RS15925] [protein=SDR family oxidoreductase] [protein_id=WP_042225860.1] [location=3487598..3488338] [gbkey=CDS]
MKGKVAIITGSSRGIGAAAAKLLAGRGAKVIVNYASNASAAENVVASIRENGGEAAALQADARNAEQMNGLVSETASRFG
TVDILVHNAGMNFVQKSFEDMSWEEFIQKTNDELQAAFVSTKAVLPFMKKQKYGKLVYVSSALGNHAAPLFISHGTSKGA
LNSFVRYIAQEFGGAGITANIVSPGMVDTDATASIPEAFKQQQATRLPLGRIGQPADIARAIAFYASEDSSYLTGTYLPV
SGGGEM
>lcl|NZ_CP026520.1_prot_PC41400_RS15930_3145 [locus_tag=PC41400_RS15930] [protein=oleate hydratase] [pseudo=true] [partial=5'] [location=complement(3488392..>3488604)] [gbkey=CDS]
SALTDDSENFAFIGHFT*IPDDVVFTVEYSVRTAQTAVYKLLDIDKKIPGMYHGNHHAKVLFDSLVAMVR
>lcl|NZ_CP026520.1_prot_WP_042225862.1_3146 [locus_tag=PC41400_RS15935] [protein=MFS transporter] [protein_id=WP_042225862.1] [location=complement(3488607..3489818)] [gbkey=CDS]
MDNPYSFRSLQVYAVTILFYCIVYMVLMILPFYAITAGAGEMEIGLIMGTTMFASMICRPIAGTVIDRYGTRKVFVLALL
LFTVSLAGYFIPNLWLFGLVRAVQGIVAAFFSTAMEIITMDLLSEKMRAQGLSLYSLATMIPSTFGPTIALLLKDWLPMI
WIFGSLFMMSMGIFLFGLSLSRQMKPSAAAPPAEIPDDPAVGPLSRKGVWKSRTLLLSSAVMLLASVANGAIFTFLPLYL
EKQGLKFAELYFLIQTLVLVLTRFAGRKYIPSDGSFPGRILFPALALASLGSLILGTSLSPAVLAAAAVFNGIAFALLYP
ALLTYVSFSVPHFARGLLIGLFIGAADLGFSLGALAMGPVASSFSFPVMFITGGALCLLAGFAGLGFRVRRTENGKNRTA
AGL
>lcl|NZ_CP026520.1_prot_WP_042225864.1_3147 [locus_tag=PC41400_RS15940] [protein=cysteine synthase family protein] [protein_id=WP_042225864.1] [location=complement(3489811..3490911)] [gbkey=CDS]
MSTILDTIGNSPLIRLNRLVESADSEIYLKMERTNPGGSIKDRPALYIIEEAERRGWLRPGGTLIESSSGNFGISLAMIG
AAKGYRVIVLVDPKTTPVNLAMMKAYGAEVIVVTEQDDSGSYHKTRIALANKLHREIPGSFRPDQCFNPWNGEAHYVQTA
KELMDQSGHGIDALIVTVSTGGQIGGLSRYIKEYAPHVKVIAVDAVGSTIFGGKAHAYLLPGMGLSWTPTNVDDLRRIDE
IYKVPDEDAFLMCRVLAKHEGILGGGSTGAGMVAAWKLAQLMGRGKRIVCVASDNGERYLPTIYNDDWLQERGILTDIGL
QEMKVRVRELPPYSRNPAETANYKPELAEQLDSPSRLISRELFRHG
>lcl|NZ_CP026520.1_prot_WP_042225866.1_3148 [locus_tag=PC41400_RS15945] [protein=PepSY-associated TM helix domain-containing protein] [protein_id=WP_042225866.1] [location=complement(3490939..3491490)] [gbkey=CDS]
MKTRKYHLWVGLIASIFIFVEALTGILLVHTSSMGFNKQVQTYEAPAASSTGQQPLNVAEAVLKASQSGAFDLSQARLVM
NHTLGDGHHGTAGDYKVRLRDDDQTLYVIDLYGNVIRKEVNNVNNFVRDLHFGELYGKDITWFVDIAAISIMFLTATGIV
LSIRVLKAGSIRKSKRKKAEAAG
>lcl|NZ_CP026520.1_prot_WP_042225869.1_3149 [locus_tag=PC41400_RS15955] [protein=GNAT family N-acetyltransferase] [protein_id=WP_042225869.1] [location=complement(3492228..3492764)] [gbkey=CDS]
MKDNLESERLLLRTLTEADAPRLFEIWSSPDVTKFMNIDPFTEVKQAADMIAFLNGLALENKAVRYAVIEKEADAIIGSC
GFNELDYGQRKAEIGYDLDPAFWGKGYAPEAIGCLLAYAFGELGMNLIEAKVEPENVNSIKVLQKLRFTFAGTLSSGEDA
DSGPSSPPLNLYTRSRED
>lcl|NZ_CP026520.1_prot_WP_042225870.1_3150 [locus_tag=PC41400_RS15960] [protein=2OG-Fe(II) oxygenase] [protein_id=WP_042225870.1] [location=complement(3492921..3493634)] [gbkey=CDS]
MPDNIACRIAGLDWDSIRLSLDEHGYAKLPPMLESSQCEELIGGYGEEQLYRSTINMARYRFGLGEYKYYRAPLPDVIRE
MREAFYPPLADIANSWLKRLDKETLYPDSLSGFLEQCRREEQTRPTPLILKYEAGGYNCLHQDVYGSVFFPFQVVFALNQ
REQDYTGGEFLLVEQRPRAQSRGHVLTLERGEGLIFPTTHRPVSGSRGYYKTTLRHGVSTVTTGTRYSLGLVFHDAK
>lcl|NZ_CP026520.1_prot_WP_042225871.1_3151 [locus_tag=PC41400_RS15965] [protein=EamA family transporter] [protein_id=WP_042225871.1] [location=3494118..3495155] [gbkey=CDS]
MWFMYASLAAVSFGLRGILYQWTSQRRTDRNVLLFGVYLSGALISFAVNLFVNQAWTYGVWLGVPMGLFSFIANASMYKG
YSVGRASLIALFTGLPPVVVATLAYFLWGEALGIVQLAGFCIVILGLLVIRYSHDLKLGQLQGIQWGLLTMLFFGFTDLS
SKQATLSAANTLPLLTVMYGTGTIMFACMYLLSRLKVPAETGQKIVASETASVSTNDRELGYGHDAQHAQISRHSDEDKG
LPGSEDDLRTHPATGGTDNPASPEWSMKRTVLWGMTVGITNLAGMLFIIPAFRGGVTGIVSAISAMSVVLVLLYAQFYLK
ENISRREACGMLLALAGILVVRLAS
>lcl|NZ_CP026520.1_prot_WP_084706496.1_3152 [locus_tag=PC41400_RS15970] [protein=helix-turn-helix domain-containing protein] [protein_id=WP_084706496.1] [location=3495343..3497661] [gbkey=CDS]
MYSLKRIPFNRRSVIVTWLISYITVLLVPVLINGVIYMVTWNVVETEVNRSNEAVIKQMEQAIDNHLKGIERLSVEMALN
KRLAGFIHAGSPLTDDDHYELIGIAGDLGVYKVANDFIDQIYIYYKNSDTVISSRERTDSRVLYETLRENEGTDYGEWKS
FFDKRFIQEYAPVTLSENGSKTGAVLYAKSVILDNPDLPGGVILFVIKESKLLANLAPVHNASVAVLDEKNRLLASSGTD
QKTDYPAYGELSGKSGLVYKKSAGRNLAVSYTTSEQTGWKYVSMMPAEIFDEKMEHMSVLIFVNLFLSLLIGGILTFLFL
KKNYSPIHVLIRNLSTKSGLAFEEGSNEYGFVQKALNNAFAEKENVSRRLMQHRDAIRSHFLQGLLKGRLDPNVPVHESL
SAHDIGFPYEHYAVLLFRIDQFGKWDEPGGGDAKKIRQIQFLLMNVAEEAAVAGRCRAVTAGIDDMQACILNISPEADEG
EIRRLAQQVQAFLLDHFQVRLTIAISGVHAGLATVPFAYQEALAAMEYRIVMGSGEVIAYGDLPGGDEFEEAAFYSYPMH
VEQQLINFVKAGDFGQANALVEQVIERNVVQKTLSVPLSRCLLFDLVGTLLKALDEIGLPGKRELIRELNPAERLAGSET
LQEMTAGMREVLSRVCGFIQESRKPEHDQLSRQVAEYVRQNYSSENLNISMIGEAFGLTPSYLSKQFKAQTGEALLDVIQ
KVRLAEAKKLLSQSSDPIIEIARKVGYSDINTFNRIFKKFEGVTPGKYKEIL
>lcl|NZ_CP026520.1_prot_WP_042225872.1_3153 [locus_tag=PC41400_RS15975] [protein=extracellular solute-binding protein] [protein_id=WP_042225872.1] [location=3497866..3499491] [gbkey=CDS]
MVRHAGRKNSRSGVIVILAASVMLAACSAGGRGGKEEKESAVASEPGKYPIRAEETLTSWEPMDTNLSAFVTNLADSPFG
KQLEKETGIKVKYTHPADGQSKDQFNLLIASNNLPDVIEYNWSGGNGSYPGGPEKAIADKVIIPLNPYIDKYAPNLKKLL
EQDKELDKMIKTDSGNYYVFPMIRPENGLVFRGPMIRKDWLDELKLDVPVTIDDWYTVLKAFKERKGAASPFSALYQNEF
NIQDAFIGAYKTANRYYIDDEGQVRYGPVDPQFKEALTLLRKWYAEGLIDKDFALNTDSKSLDTKMLNDSTGATVHLLSG
GMGRWMDTGKKKNPNFQLVAAPYPTLTKGERPFIGQRDFKYNPAASKAVSDASKNPELAVRWLDYAYGEKGSLLFNFGTE
GDSYTLNNGTPAFTDKIAKDGKYTLQQMISQYTKPNGPYLGDSRKNFNTFKQQDEALKIWSETDAAKHVMPQFITPTVEE
SKEIAKLNTAITSYKEEMFIKFVMGKEPLDKFDEYVKRIKEMGIDRVTKIYQDAFDRYNKR
>lcl|NZ_CP026520.1_prot_WP_042225875.1_3154 [locus_tag=PC41400_RS15980] [protein=ABC transporter permease subunit] [protein_id=WP_042225875.1] [location=3499587..3500543] [gbkey=CDS]
MAAINQEALPYPAVKKRTRTGLLLDIRKNKLLYVMLLPVLLYYIVFHYAPMYGAIIAFKDFSPRLGIWGSEWVGFAHFET
FFSGAYFWRTIKNTILISLNELVFGFPAPIILALLLNEVRNSVFKRTVQTVTYMPYFISLVVICGIIKDFTASSGVINDI
IAFFGGERLTLLLEPEWFRTIYVSSGIWQHIGWGTIIYLAALTGIDQEQYEAAKMDGAGRWKQMLNVTLPGLMPTIIILL
ILELGRMMNVGFEKIILLYNPSTFDTADVVSSYVYRVGLQEFNYSFSSAVGLFNSAVNFTLLLGSNWISRKLNSTSLW
>lcl|NZ_CP026520.1_prot_WP_042225877.1_3155 [locus_tag=PC41400_RS15985] [protein=carbohydrate ABC transporter permease] [protein_id=WP_042225877.1] [location=3500562..3501437] [gbkey=CDS]
MNKRSFSERAFDGFNLFLLTLLMIVTIYPLVYVVFASLSDAGGLLAHKGFLWKPVGFSLDAYANVFRNPMILKGYGNTFF
VVVLGLTCNILLTSLGAYALSRKSLAYRKRIMLFIVFTMFFSGGLIPFYLTVKGVGLANTLWALIIPTAVNTFNLILMKT
AFEGIPDALEESAKIDGANDFVILFRIVLPLSLPVLAVMLLYYGVGHWNSWFNALIFLQERSMFPLQLILREILLQGEAN
ANLGASEGDVAMLTVTLKYATIIVATVPILLVYPFLQKYFVKGALIGAIKG
>lcl|NZ_CP026520.1_prot_WP_042225879.1_3156 [locus_tag=PC41400_RS15990] [protein=hypothetical protein] [protein_id=WP_042225879.1] [location=3501449..3503716] [gbkey=CDS]
MKPSVTDSLRLDLDWQAFMAEHDMEWVVKPTSWNEGAYIGNGTIGAMVYGAEHAKKRRTLRFVMGRTDVTATRTDRDDFK
PRVPIGEMNLELAGLIYNPTRLRVDLWNAELRGLLTTTCGEVCIRALVHAEETVMAIELETSEEEREARLEWVPHSEVDD
VLKFADGINVNQYIPLVMHETSREDGVTIAIQRFAGGEECGEGCVTAWKVMQLPEGRRVAFVSIQNSCTDEARKNAVEAV
RKAASADFGEWVSAHRRWWHGYYRQSFLSIPDRHLESFYWIQMYKLASAARADSVLLDNQGPWLAPTPWARNWFNMNVQM
SYSPVYASNRLAIGESMVQAVADNREHLRENVPLPYRHDSAGLGRSGSFDLRSETGEEIGNLTWLCHCLWRQYRYSMDDG
LLRDVLYPVLRGSVNYYLHLLHAGEDGKLHLPPSISPEYGSFKRLTVSDCHYDLALLRWGCETLLAADARLGNGDPLAGR
WREVLDNLTPLPVDESGYMVGRGVPLAFGHRHFSHLLAVFPLHLVSPDREEERELIARSLRHWLDSEGDLRGFSFTGAAS
IAATLGWGDAALRLLRALLGLIKPNTMYKEAGPVIETPLAGAESLHDMLLQSWGGTIRIFPAVPEEWREAAFHDLRAEGA
FLVSAVRRAGATSFVRVKSLAGEPCEIRTGWTGAVHWRLDGETACHVVQVADSRESITLPLNAGDDAILYPAGEEPELRI
GPASGKTGPVRYFGGSKPWRLHGFPFRNLLEEDRT
>lcl|NZ_CP026520.1_prot_WP_042225881.1_3157 [locus_tag=PC41400_RS15995] [protein=SGNH/GDSL hydrolase family protein] [protein_id=WP_042225881.1] [location=3503713..3504363] [gbkey=CDS]
MKEHKDRAIGRGELLLFQGDSITDTGRNRENHADLGKGYVQMIAGWFGAPCPGSGVTFLNRGVSGNRVKDLQNRWEQDCI
DLKPDWITICIGINDCWRRYDRNDPTSLETFAAGYRQLLTRTTEETEARIVLMEPFVLPVREDRKAWREDLDPKIHAVRE
LAREFGARLVPLDGLFARASTEAPASYWAPDGVHPSAAGHGLIARAWLDTMKVQFV
>lcl|NZ_CP026520.1_prot_WP_042225882.1_3158 [locus_tag=PC41400_RS16000] [protein=chondroitinase-B domain-containing protein] [protein_id=WP_042225882.1] [location=3504539..3506416] [gbkey=CDS]
MNAGNRTFSLILSLCLLLMLVPAQGYAANAKFSVPASAVTASKHDGNLPANTVDGNLSTRWSANGDGEWIRFDLGSLRKV
AYIKVAFLNGDTRTSTFDIQTSADNVTYTNVKTNVTSSLSAQLQTFDFTDVSSVRYVKLVGHGNSVNLWNSYTEVEIYGE
DGSQGGIPVSTSEGLAAALKTASAGQTIVLADGNYTVSGNNPSILIENRNGTEASPITIKAANRGKAVITGSTTFEVKKS
SYVTIEGLKFANSADKGVLLNGSHHIRLTRNQFALPARGVDTIWLQVSGANSHHNQIDRNDFGNKTDTNPLIAYEGDGQG
NISQHDVIEYNYFHDVGPWVDNGKETIRLGLSKISLSDGFNKIQYNLFENTDGEPEIVSVKSSNNTVRYNTFKTSKGGLT
SRHGHNNSFYGNFFLGDGVKSEQAGIRIYGNDHKIYNNYMENLTNSAIILDNGNYDGGTGGYPSNPSKDDLKAQWRIYRA
QVVNNTIVNSTTGIVVGSGKAFTPVDSRVANNIVKNSSGILYNEAAATNTVFEGNIGFGGTVTNKGRTSAEIWNKNPLLT
VVQGLQKLSASSPAINYAKGSYPFVQEDMDGETRSVNDAGADERSSASSFTNHPLTQAEVGPDAP
>lcl|NZ_CP026520.1_prot_WP_042225886.1_3159 [locus_tag=PC41400_RS16005] [protein=TetR/AcrR family transcriptional regulator] [protein_id=WP_042225886.1] [location=3506657..3507253] [gbkey=CDS]
MNKKQLQSEQTKKRVADAAKTLFVQKGYKSTSIEDIVEATGSSKGNIYYHFKSKEGLFLYLVDEWEREWEEKWMEKEPLY
KTTTEKLLGMAEQMVIEDLNHPLTKAADEFFNNEEKTNDVEERIAGLVNRHLKFNQVLIQQGMDSGEFKGADVEPLAMIL
ESLIMGLSQMSRKLSTAEALKLYRLAFGVFLHGVAKPS
>lcl|NZ_CP026520.1_prot_WP_042225887.1_3160 [locus_tag=PC41400_RS16010] [protein=MFS transporter] [protein_id=WP_042225887.1] [location=3507406..3508605] [gbkey=CDS]
MELLLKNRGAVLLLMLNIFLVFTGIGLVIPIMPTYMDELHLNGSIMGLLVAAFSLTQLIFSPLAGRLSDSLGRKKMIVAG
MVIFTVSELLFGVVSSPVLLFASRMLGGIGAALIMPSIMAYIADVTTTEERAKGMGFINAAITTGFIIGPGIGGYIAEFG
IRAPFYAAAAVGAVAAVVTLFILPESKPVEPPVKAEAAPAQKQSFMAQMLSSFREPYFISLIIVFVMSFGLANYETVFGL
FVDHKFGFTPKDIAFIITFGSISGAVVQVTIFGWILNRFGERTVISVCLAMAGLFVLLTLFVHQYWMIVVVTFIVFLSID
ILRPAIGTQMSKMANEHQGYVAGLNSAYTSLGNITGPIVAGVLFDRNLNFPYVSAALILFLCFALSLRAGRRGSEPGIS
>lcl|NZ_CP026520.1_prot_WP_042225888.1_3161 [locus_tag=PC41400_RS16015] [protein=proline iminopeptidase-family hydrolase] [protein_id=WP_042225888.1] [location=3508660..3509526] [gbkey=CDS]
MRHTEGYIEVPGGKVWYSSAGEGGGTPLLVLHGGPGNTHDPLKAALHVLGDQRPVIFYDQLGGGNSDRPEDATLWHTERF
IEELACVREALGLDEVHILGHSWGTMLAASYLIERKPTGVRSVIFSSPCLSAARWKEDADRFLAELPDEVQAVIARSEEQ
GTTNTDEYRDAMKEYYKRHVCRIDPLPAVMLESRGKANKDIYMSMWGPSEFCPTGSLKTYDVTPRLHEINVPSLFLCGRY
DEAAPESTFYYHSLVPRSDFQVLENSSHVGYLEEPDEYVRIVRGFLTS
>lcl|NZ_CP026520.1_prot_WP_042225889.1_3162 [locus_tag=PC41400_RS16020] [protein=SDR family oxidoreductase] [protein_id=WP_042225889.1] [location=complement(3509829..3510593)] [gbkey=CDS]
MQNLREKTALVTGASRGIGKAIAKRLAHDGAWVAVHYGRDKKAAEQTVAEITESGGRAFSVQADLGSMMGVERLFEQLDT
LTIKHTGSTSLDILVNNAGIGTQGSIETTTEKQFDELIAVNIKAPFFIIQKALPRLRTGGRIVNISSAETRIAFPASIAY
GLTKGALNTMTLPLAKQLGERGITVNTILPGYTETDINAEILSDPSVREYAVGMTALGRLGHVSDIADAAAFLASEESRW
VTAQILDVSGGIRL
>lcl|NZ_CP026520.1_prot_PC41400_RS16025_3163 [locus_tag=PC41400_RS16025] [protein=Lrp/AsnC family transcriptional regulator] [pseudo=true] [location=3510732..3511171] [gbkey=CDS]
MRETAKPDDTDRKIMNLLHTNARMPVAEIGRHISMTQPAVRERIQKLEEQGIITAYKAKFDPGKLDWAYRHLCCLKPSNA
GSLFVIANLPRRLPSCIG*AGSSIIC*KRGANRWSRLPRSRTR**FSEYPIR**SCTFMRRTKVHF
>lcl|NZ_CP026520.1_prot_WP_042225893.1_3164 [locus_tag=PC41400_RS16030] [protein=helix-turn-helix domain-containing protein] [protein_id=WP_042225893.1] [location=3511370..3512368] [gbkey=CDS]
MTLDIVNSEVHSLFDQFTDLIQGQSRLRTWEQHITIPHGVGQGSILRTRIRPDMEIVITDVTYAQDMKLRIQEACPLFEL
SYCLSGDVYCEWGGKESYTSHWTGNVLFFEDELVYEEKKAGVRNHMVEIRLSPKELFHYAADFTEKQRMESWLQRHKGNI
DKYPNTPEIHRCVSDMLNCTYDGAMKRLYMESKAMELIALFGEVEGYEAEIGKTFLNRDDLAKLDQARELVIRHFEQPLS
IRELARKVGLNEFKLKKGFRERFGMTVFELVRKQRMEKALYCMEVRRMNVGETAVSVGYSNFSNFAAAFHKMYGYNPSEY
LKMLSHRDAETE
>lcl|NZ_CP026520.1_prot_WP_042225894.1_3165 [locus_tag=PC41400_RS16035] [protein=MFS transporter] [protein_id=WP_042225894.1] [location=3512567..3514012] [gbkey=CDS]
MKDLTTVCTREPSAAPHADPRRWMALVLILLPTLLISLNNYMMQVALPMIQGSVHASFAEAQLIFSGYSLGLAVALILGG
KLGDLYGRKRMLRIGVLGFTCMCLFGGLVSDPTLLIVIRIIQGLSAAMIQPQVLSIMQSIFLPREKGLVFAIYGAVIGIG
FTLGLILGGFLVDGNLFGLSWRIVFLVNVPFGLLVMMGLPMVPESRGQSEPSIDWIGIALVMSGLFLLIYPLTIGQKQGW
PVWIWGCLLLSLVLLFLFVLLQKRREETGRVPLMKLSMFKIGSFRIGIITEVVVYLSMFSFFFVLNYYLQAGLQFDLKST
SLVFLPIGLGCFAASLMSAWLVKRWGVAVLKTGMLTMGASLFILIWSLHADAVHLLHFQNVLILSVYGLGLGMATTPLAN
TILNRVPAVSAGTGSGLFTTFMYLANALGVAGIGILFSSALNSPLAEAGLEDYIRAFSVTTAASGVLAFAACACISMLKK
E
>lcl|NZ_CP026520.1_prot_WP_042225896.1_3166 [locus_tag=PC41400_RS16040] [protein=esterase-like activity of phytase family protein] [protein_id=WP_042225896.1] [location=complement(3514110..3515564)] [gbkey=CDS]
MKKNWLIAASAALTLVSGTGSAFAADLKGSTADSVTYTNNGETVAPSVPAVNIDGSLYLPFRAISEATGKYVDWDEFRSR
VSLTDKPALTGKYKLNAPDLAEGLKMGIGSSLTHLPGDPDNVFYTTADRGPNGEVEVKGTMVRTFPLADYTPTIYKIEIK
DGQINILDQFPLKLNGINPATGKANISGLPNIKGRDEAPFDAKAEKELAYDPYGLDVEGLAYNPKDDTFWISDEYGPSLV
HVKRDGTLIERIVPKGWAAQVSTPLVPARETLPAVYNKLRQNRGAEAVGITPDGRYMFMAMQSPLRSPNKESDNSRQLRI
IKFNLQTLEPVAEFAYLAEDAKAFKGLKQADIVISDLVVVDENTLLIDERDKNAGDKAQLKRIYFIDLSNATNILGKYDD
TKAAGKTLEQMSPAELKDQKILPPTKRTLLDLVPFKYPYEKVEGVSLVNGNTLVIVNDNDFGVDSSSPENGTELWTFKLP
YTIK
>lcl|NZ_CP026520.1_prot_WP_042225898.1_3167 [locus_tag=PC41400_RS16045] [protein=MFS transporter] [protein_id=WP_042225898.1] [location=complement(3515878..3517071)] [gbkey=CDS]
MFKTWKIYMLALISFLVGTSEYIIAGILDKISASLDISLVAAGQLITVFSLVYGLGTPFLIAWTARWERRKLLIYSLGLF
VVANLLAFILPGFAAFVGARVLMALGAGVVVVTALTVAAKIAPSGKQASAIATVITGFTASLIVGVPLGRLVAATYDWKT
IFAGIGALGLLAMFILSVTIPKTEGDAPVPLREQMALLKQPRILLALSVSFFWLGGYSIAYTYISPYLLTITGMNEKMLS
AALLAFGIASLVGSKFGGYSADKWGVYRTLMSGMTLHIITLLLLTLAAHSPVVVFAVLILWSFAAWSSGPTQQYNLVTMA
PESSGIMLSLNSSVMQLAMAAGAGIGGIIVGSISLNAVTWIGAAGVAAAIVINFVSFKLSAAERHKPEREHPVRLNA
>lcl|NZ_CP026520.1_prot_WP_042225900.1_3168 [locus_tag=PC41400_RS16050] [protein=metalloregulator ArsR/SmtB family transcription factor] [protein_id=WP_042225900.1] [location=complement(3517231..3517497)] [gbkey=CDS]
METNQAVKVHRALGEHTRYKIIQILTKESNLCPADLESRLESIALSTLSHHLKQLSDCGLLHSQKKGTYIYYSLNEETVK
KFAPYLIE
>lcl|NZ_CP026520.1_prot_WP_042225901.1_3169 [locus_tag=PC41400_RS16055] [protein=glycosyl hydrolase family 18 protein] [protein_id=WP_042225901.1] [location=complement(3517709..3519268)] [gbkey=CDS]
MNWQSIRSQSPKQVLFRILLIVSLLVSLLPFLGLNTGQASAAADVSAAAAACTDAAWDSTKVYTGGQKVSYNGKVFEAKW
WTQGETPSQTNTWGSWKYVSDCTGGTPDTTAPTAPAGLTLTSKTSTSVTLSWNASTDNVGVTGYNVYNGSALTVSVTGTT
ATVTGLAANTAYSFSVKAKDAAGNLSAASNVLAVTTSSDGGGTTPLTKINGAYVASWHFPAVNTVPANKLTHIFYAFADV
ANNGVSGGDTSQIAQLVTLKQKNPDLKVLISVGGWGRSDGFTSAASTDANRTAFANSALQYIRANKLDGIDLDWEYPTTA
DKQNYTLFLQKLREVLDAGSAADGRTGNKKYEITAAMGASEYGVQGIDLGGVHPYFTFINIMTYDMQVGANTHHTNLYTS
SLNSNYSVDNAVKLYKGRGVPSEKIVIGGAFYSHGAGDYTYDELKASYINKNGWTRQWDDVAKAPYLTKGSSMLSYDDAE
SLTQKVNYLKSNNLGGIMFWEYGQNLGADLLDAIYTALK
>lcl|NZ_CP026520.1_prot_WP_241688668.1_3170 [locus_tag=PC41400_RS16060] [protein=CPBP family intramembrane metalloprotease] [protein_id=WP_241688668.1] [location=3519765..3520661] [gbkey=CDS]
MIKPSLETQGGRASTGLTRDTGGRIHLVWRILLGWLSFVVALGVAVAVTLAAQNYGFPVLAQQIILAVVTTGIAVPLIYL
LRRYADNRPWSGLGLSPLPSGFRYLLLGAGFLIAMMGIALLAGTLAGWVRVVDVHLPVETLLVILINIPIAFFYEAFPEE
VTFRGYLYGNLSERFNRWLALLFQLILFVLAPVALTAALVAAGVGTWDLITIDYVVNLLAFGIALQLCRIFTGNLWMCIG
FHLAWLEMVRFVIVPSDSALIEVEYLSPYGSYLVSLGSVTIGILVLLYWTYRNRNRQK
>lcl|NZ_CP026520.1_prot_WP_042225902.1_3171 [locus_tag=PC41400_RS16065] [protein=NAD(P)H oxidoreductase] [protein_id=WP_042225902.1] [location=complement(3520792..3521367)] [gbkey=CDS]
MKVLTVVAHPRTDSLTMAVTDRFTRGLQDAGHETEVLDLYRSRFNPVLNEPDEPDWTDSGKQYSAEVMAEIERMKRHDGL
AYIFPLWWYGLPAMLKGYIERVWNHGFAYGPQKLPHREVLWLSLAGATSEQLAKRNYDQMVNHQLNVGLAEYAGIPSSKV
EFLYDTLSSNPDHIQDLLQQAYEHGLHYGRE
>lcl|NZ_CP026520.1_prot_WP_042225903.1_3172 [locus_tag=PC41400_RS16070] [protein=helix-turn-helix domain-containing protein] [protein_id=WP_042225903.1] [location=3521763..3522101] [gbkey=CDS]
MPAKPKNQRTYNIPVEATLDVIGGKWKTVILCCLTEGPKRTSELKKDIPAITQKMLTQQLKELVEDEIIIRTVYNQVPPK
VVYELSPLGRSLESILDLMCDWGDQYIKGYIK
>lcl|NZ_CP026520.1_prot_WP_042225904.1_3173 [locus_tag=PC41400_RS16075] [protein=family 10 glycosylhydrolase] [protein_id=WP_042225904.1] [location=complement(3522212..3526603)] [gbkey=CDS]
MDRTFRGYRPTLSLFLAFALLLGLFPSISASAYSTPSSSLLAGPATSVTNGTYNSIALTPNPVTVTPSSLQQFAASAYDA
NGSVVPLQAGDVSWSADSAIGTIDSNGLLTAAAGGGGFKAGYVTATYGGLSTQALVMVGSKATAVIEDFETVSNGKALLS
AGTAGAVGTKSSISLAERPEPVLYGSRSLKFTYDMRGTAGTSAAFVSLRNPDTGALDRPIEGTPKKIGVWVYGDESSHWL
RARLRNQAGVSFTVDFTSSTAFNWKGWRYVTAEIPSTQSGPFKFMDLYLVETKDTNKNAGVLYYDRLSAIYSDTEITGLD
LTGLTPMKTGQSKTAEVYMTKANSTSPQKVESGVRFYSSNPDVAVVDGSGRVSALKAGNAVITALYSDSQPAAFEVNVTV
DEPQVLGIQAFAPDQLEAGRTGTPKVFAVYANRADSLEVTKEARFVSSNPSVAEISVDGKIVAKAPGSASVTVTFAGRQA
VIPVTVIEPVPVLTKIQLSDMKSMNIGAVRQAKVLATYTLLDVPQPPAAAISGVTYTSSNPAVAEIDAGGTLTAKSIGVT
TISASLNGKTDTYSLVVNKETGAPKHEMRAAWIATVENIDWPVKGTYDAEQQKRQLIHLLDELEATGINTVIYQVRPTSD
AFYRSALNPWSAWLTGTQGKDPGYDPLAFAIQEAHKRNMELHAWFNPYRISNDTDKSKLADSNPAVRHPDWVVSYGGKLG
YNPGNPEVKRYIIDSIMEVVNNYDIDAVHFDDYFYPYPVTGMDYPDAAEYAAYGSGMSIADWRRSNVDSLIRQLSAEIKK
SKSYVQFGISPFGIWRNKANDPAGSETNGLESYSAIYADSKKWVDEQWIDYITPQLYWNINYSPAAYDKLIDWWKRQVEG
KHVLLYAGHGVHKIGADDPNWLDPDQLPNQILFNRNFADVKGSMFFSAVQVLENKLGFRDRLKTDLYKYPSLMPEMSWLS
KSLPAAPAVTAAVEGSGIKLNIQNEAGSTAGTYVIYRSAGTGAPDTSNPAHIWKQVKVGPAGVQTVVDNTVKAGTTSTYV
VTALERGSSLESRGTTLAATAQSSTGSDGSGSGSGGTGSVGGSGPTNPSVPTPMPTPRDGVQVIEPADFKETDGKVTVSV
PAGIREVHLPADTGTMIGNSRPLLLRFANVSVTIPNPVIRQWQDLTADTGAKLVFKAEPVEGGHLQALVQTAENKLGAKI
RTRGLYDFRMEVRAGGKTAAVMDRFSAPLEISFQSAEGSDLSRTGLYHLAGDGTLQYIADQRPSGSGTIMASIRHFSTYG
LLEVSKSFQDVNAGHWAYPAISELSRKLIVTGVTKDLFSPQRQVTRAEFAALLVRALGLPAKAATPFSDVKAGDWFAGEV
AAAFDAGIVTGMTDSLFEPGGSVTREQMAAMLIRAYEAKHGKLSAESPSVSFTDAGSIHEWAKGDVAKAAAAGLLNGREG
NKFAPQAVLTRAESAQAVYNLLK
>lcl|NZ_CP026520.1_prot_WP_042225905.1_3174 [locus_tag=PC41400_RS16080] [protein=VOC family protein] [protein_id=WP_042225905.1] [location=3526839..3527252] [gbkey=CDS]
MSQVKPSAVTPFLTFTGQAEEAMNLYTSVFENSEIIRITRYGPGEHGEEGSVIHASFSLNGQAFMCIDSSVKHDWTFTPA
ISLHSFFETEEEIRRVYEELSRGGQILMPLASYPFSKMFGWISDRYGVTWQLQLAGS
>lcl|NZ_CP026520.1_prot_WP_042225906.1_3175 [locus_tag=PC41400_RS16085] [protein=GNAT family N-acetyltransferase] [protein_id=WP_042225906.1] [location=complement(3527576..3528100)] [gbkey=CDS]
MNAPHSGKSASFEISVRPWSENDLTLLKKMNTAEMWAHLGGPESEEALSSRHQRYMKAVPGKVRMFTILFGDEAVGNVGY
WAQHWRDQDVFEVGWGVLPEFQGHGIAMTATSAMLDILRDDVPGAIVYAFPGVSNFPSNAICRKLGFTLAGETEVEFPKG
SWMKSNEWRLEFPG
>lcl|NZ_CP026520.1_prot_WP_042225908.1_3176 [locus_tag=PC41400_RS16090] [protein=S-layer homology domain-containing protein] [protein_id=WP_042225908.1] [location=complement(3528280..3529152)] [gbkey=CDS]
MKNKKNMIAALTLGCSLLAASSASAFSDVQGADAAVITSLQSKGVIQGVSEDKFAPQGKLTSVQGVHLIVKALQLKAADV
TLQNPNAPWYAESLDIAAKNGLPVELAADPSAELTREQFAQLLYKGIQATGDYPLIKMYIRVADEDQMTPLYQGSIQNLL
LMKVATLDDKSLFHPQDRITRLEAAGMIAKAIEFTDRHKEQTDNQENQEVSFTIEKVNADINKIVLTRREQPSPGYGIAI
TNIEMNDSGHAKIRYKLTSPEPGKMYPQVLWTSKTETYLSSKYKVEIQAN
>lcl|NZ_CP026520.1_prot_WP_042225911.1_3177 [locus_tag=PC41400_RS16095] [protein=GreA/GreB family elongation factor] [protein_id=WP_042225911.1] [location=complement(3529237..3529674)] [gbkey=CDS]
MNLSLTGSRHQLINQLVYFDEELHQFLELYFPSPTKSRTYTEERLQEYSEALTGLISEFSEELLHSKVLIGSKVQIRYLE
DNESDTYSIVFPSKTDPSSNRISFLSPIGSHLLLGTREQTFTIQTPSGEYSVRIEDIRYMNAGDV
>lcl|NZ_CP026520.1_prot_WP_042225913.1_3178 [locus_tag=PC41400_RS16100] [protein=sensor histidine kinase] [protein_id=WP_042225913.1] [location=3529897..3531246] [gbkey=CDS]
MLYYFLALSLASLILLIPHLHKETNRWAAFFLFSASIGGLTGLLRQKGLTDLAYASEFINHTLTPYGVLVFSLVYSGVLE
NRPRARRMVKGLLLLPVPAMLIYTVSRDAFAIHYGPLLLWTAPYYLGACSLLVLSLRREGNPVRKRNRLITTLIIVPTLL
GVLLFINVARVFSPEFDFFSYVSVFMIYSFAVALLCVFVYGVLGIRLRFERDSLDTTMQAVTSGTTLLNHSIKNELGKIA
LSTVNLQGSLDEKDEEAKEHLRIIAAASDHMLEMVGRIHNRTKEIALKERPVPFDELVMEVVRRHGQLFARQGIAVQTVF
DWRPIVRFDPVHLTEVIGNLLGNAAEAMPDGGTLTVTLASVRHGAVLSVADTGAGIPSDRLSRVFEPFYTTKTKAANYGL
GLSYVYNVMQKSGGSVEISSKPAAGTCVRLYFPRNKIVQLNGEKPDGTN
>lcl|NZ_CP026520.1_prot_WP_042225914.1_3179 [locus_tag=PC41400_RS16105] [protein=response regulator transcription factor] [protein_id=WP_042225914.1] [location=3531233..3531871] [gbkey=CDS]
MELIKVLLVEDDPDWIKAMTSYLNKEDDLLVVGAAVHPDEAVSMAAKLDFDVVLMDIQLTDRGLDGIYTAVEILELRPAK
IIMLTSMSDEHVMTEAFTAGAVNYIEKTRFREVPHAIRSAYRHPAAMDALLKEFSRLKREEQLKKLTTAEREVYELIEEG
YTQSQIEKKLVKAESTLKNQVNKMLKKLGVKSSKEAVEKVRRRGISRDKQSD
>lcl|NZ_CP026520.1_prot_WP_042225917.1_3180 [locus_tag=PC41400_RS16110] [protein=thiol-disulfide oxidoreductase DCC family protein] [protein_id=WP_042225917.1] [location=3532031..3532513] [gbkey=CDS]
MGRRKKASQKDSVILLIDGACILCHGITRFVIRRDPSRRFRFAAIQSEAGSRLLKTQGLAAGDPDTFVLIQDGRAYTKSS
AALRVFRSMPGVWPVLGLAAIVPVSVRDRVYAWIARGRYRWFGRRENCLLPDLDIRKRFIDEGRETDGMESEERPDAKEE
>lcl|NZ_CP026520.1_prot_WP_042225921.1_3181 [locus_tag=PC41400_RS16115] [protein=DoxX-like family protein] [protein_id=WP_042225921.1] [location=3532517..3533473] [gbkey=CDS]
MAAKPIYVELDIQTDMETLWKATQTPELHQQWDLRFTEITYMPRSGDREPQRFHYRTRIGFGMDIAGTGETKAKTNLLTG
ERISGLKFGSPQRISLIRSGSGYWKYSPNGGSVTFATRYTYDTRFGAAGRWFDRCLFRPLFGYATAWSFDMLRIWLEKGI
PPSVSIQRALMHYFSAGLLALLWVYQGLVPKLLFPGGGELAIMQAAGWFPGGEKPALALLGASEIGIGLLAAAAHRQAGV
YIVQAALLLLLPAIALARSPELLQAPFNPLTLSGAMAGLCLLAFWSRRELPQAGRCLRKPRGSGKEEEPKWHPYTSGR
>lcl|NZ_CP026520.1_prot_WP_042225922.1_3182 [locus_tag=PC41400_RS16120] [protein=DUF4166 domain-containing protein] [protein_id=WP_042225922.1] [location=3533446..3534123] [gbkey=CDS]
MASIYERALGEEFGRLHPRIRERFGFGSEDGVASIGAGVMHRIWASKWTALPLLIGTSRHIMFPESGTEIPFTIENYAYR
DGYGRETVTWNRHFTFRSKVRHFDATMIYSSRRRRIVDYLGNKQHLAVDLELEAAPNGGIRIRSSDQRFYEGWLQFPFPA
GLTGTAEVCEWFDDAEDAYKISVNVSNPVFGPVFGYQGTFRAQLVPCGRTDIPDHAKPLREELRE
>lcl|NZ_CP026520.1_prot_WP_042225923.1_3183 [locus_tag=PC41400_RS16125] [protein=DoxX family protein] [protein_id=WP_042225923.1] [location=3534416..3534949] [gbkey=CDS]
MNWLRNNVYASGVLLLLRLYIGWKWASAGWGKITGGFDASGFMKGAIAKPVTDKATNELLYPTYHAFLESVALPGVKAFN
FLVPWGEFLVGLGLILGGLTAVAATFGLLMNFMYMFAGTVSSNPWMILLGFFVAAAGANAGRFGLDYYLLPYLASKAGRK
RKAVVFDKDNKPAFEHG
>lcl|NZ_CP026520.1_prot_PC41400_RS16130_3184 [locus_tag=PC41400_RS16130] [protein=2-oxo acid dehydrogenase subunit E2] [pseudo=true] [partial=5'] [location=<3535418..3536197] [gbkey=CDS]
VHVRSTGIHLSDNPPTPFGQTEAETVGEAEDFIKVTPVRHTIAMRTLQSVREIPHALTSIETDVTDLVKLRTKLKDEFIR
REGVNLTYLAFFIQAVVSAIKDYPVMNSVWAVDKIIMKRNVHISLLIGTEDSVLAPVIKFADQKSVSGIALEIASLTQKA
RTGRLTLDDMQDGTFTINNTGAFGSILSSPTINYPQAAILTIESIVKRPRVVDEMIAVRSMANLCLSSDNRILEGRVTGQ
FLHRVKENLEAFHPESQIY
>lcl|NZ_CP026520.1_prot_WP_042225924.1_3185 [locus_tag=PC41400_RS16135] [protein=LuxR C-terminal-related transcriptional regulator] [protein_id=WP_042225924.1] [location=3536353..3536973] [gbkey=CDS]
MREKNKPIHVFIVDNSSGKNPAIADYLSQSDQFQIAGIETGMKSASFTHLGVLPEPFVVVMLMSDSPHCDILWIKKMKKT
YADLRIIVLSEYNYVLFFHTLMEAGVNNILNMDTKPEQVAQVIECTLDGHFFLPYSFYHMLNHISSVFTEVEIQTLELIA
LDYTNAQMAKELNVTIRTVESRLTKIYDKLEVSSRRGAVKKIMQMK
>lcl|NZ_CP026520.1_prot_WP_042225927.1_3186 [locus_tag=PC41400_RS16140] [protein=amino acid adenylation domain-containing protein] [protein_id=WP_042225927.1] [location=3537231..3538790] [gbkey=CDS]
MYRSDEPKFLHDYFLKFAEKTPESLAIVHHSEKWTYAGLADKSRDYAERLKIAGLRTGDRVVLELDPSAEAIAVMIACSI
RGFVFIPLNPDTPQERLESILTRTGARLHIQKGRTRKLSPQAEARQAYLEEGHLLVPEASQAADNLSTDSKLLDTNLVYL
IFTSGTTGQPKGIMMTHRAVVSYFDGMNAQCALPQNTRVGSVSSLQFDFSLVDMGLAFGTGGTLVIVPRSLIFQPRRFLE
HLHQQRVDLMSGVPSIWKNVLAHERQPVEDIHLHSILYGGEYCSARDLRKLQSLFGLKRIINCFGQSESIACSFKHLPLP
LPDTEAISVGSGMHNTEMLLVDESGTIIREPGITGEIYLRGACLFSGYWLEEELTSSRLVPHPETPCTGEKVFKTGDLAY
MDENGEFYYVSRTDNQVKIMGYRVELEEVERRLLSHPKVDGVCLIVHEGNKTELHAFMVLTDQENTQTAQQQLRAYCAEM
LPSYMIPSRFSFLKEFPLNTNGKVDRKALKGMQIPTLTR
>lcl|NZ_CP026520.1_prot_WP_042225928.1_3187 [locus_tag=PC41400_RS16145] [protein=acyl carrier protein] [protein_id=WP_042225928.1] [location=3538805..3539065] [gbkey=CDS]
MTKEDVYAKFVEHIKKTNQVEELQIDENDNLLEHGYIDSLSMVDMIVFLEEQVGREIVIEDYDIRKFYTLKSIYETFFQD
HLVSSN
>lcl|NZ_CP026520.1_prot_WP_042225929.1_3188 [locus_tag=PC41400_RS16150] [protein=cytochrome P450] [protein_id=WP_042225929.1] [location=3539079..3540245] [gbkey=CDS]
MRSEATPLVIDYMSTEFRQDPWPLYTALRQEEPIHWSEKQQCYFLSKFKHIKQILLDTENFTVEHPFRTTRHLFGSTIID
MDGKKHSERRPIMSNQFKPSAMQKGMEDAVRQVIKRIVDGIPSGREVDFIQDVAIPIPMTIIMEAIGLPAKDAMWVYHKM
RPIILHLDNPKSHFQAALEASDELYAYIENFVRGGTSRGMIAHFRASVEKGQWTQEDMNRHILLLLSAGSETTACSIANI
MAILLDKPEELARVMEDGEYLKSAVQESLRWQPPLHTTTRICKKDYELDGVVIPKGKFVILLLASANRDEEMYENPGQWI
PSRKEKANLSFSMGSHNCLGFNLAKLELEETFLQLFNRFSGIKLAQSERPVIQGQTFRIPQKLLLTFS
>lcl|NZ_CP026520.1_prot_WP_042225930.1_3189 [locus_tag=PC41400_RS16155] [protein=hypothetical protein] [protein_id=WP_042225930.1] [location=3540280..3541200] [gbkey=CDS]
MVGISHVHYVLGDYKPLTEIPEMDTSKENLAKFRINGVRQYCGSDKTMYQLALETATATLEKSGLDKSSVGAVFFATGNH
HCKELHEEDFAGRLLDELGLDSAFFYGVFMQYCASVAAALQLAAQFAEANGTNVLLVSADKILDYGPIVRIIPGVIGLHS
DGAVSCIVTDRNPQFTMTPFQTLQDSSSWRLMYGQEKKGLQDSFLTNSRLVISRVLEANNQPKESIHLVVTNNFNFTISK
NLSVYWEIPYEKFFSLNIPRTAHCPAGDVFINLLDMNTAGIISRGDSILTFFPAPHSWMGTLLTKN
>lcl|NZ_CP026520.1_prot_WP_042225931.1_3190 [locus_tag=PC41400_RS16160] [protein=aminotransferase class III-fold pyridoxal phosphate-dependent enzyme] [protein_id=WP_042225931.1] [location=3541214..3542395] [gbkey=CDS]
MSLSVFDEKSIYGADYYLAGSKNEFLILEDREGRRKHYIDCMSAYASVNFGHCNPDIAPFRELRSDIAACFYPKEAEQYA
QWLLGKLKAPGYELLFQVGGSMAVSTALSIGQRVREGKIAYLRGSFHGLGLDALAVTSVHKNFALQQTSLLGELEKHCIE
IEPGQTDIPWETISTFIYEPVQGANGYIPLPAKWLTDTAAEARRHGVIVVSDEIQCGFYRHGHLSVSCHLGIEADVYLFS
KSMTNGLYPFSAVVYRGSLKESIPGSLFLAHTFQTSSLGCFAAMAVAEYIDASRPEELCGEIEALIREHFPFELPLLRDV
HLTGPTLSFEVEGVSGKSIVQQCLERGILIFTGGPAGQRIRFAPPLTIEKTNLVYAIKQLADVLTHAGAVHTL
>lcl|NZ_CP026520.1_prot_WP_042225934.1_3191 [locus_tag=PC41400_RS16165] [protein=RraA family protein] [protein_id=WP_042225934.1] [location=3542417..3543103] [gbkey=CDS]
MITEEKGIDLSTIEQVRRQFANIGCAQLNDASQKYTVPFPTTFQNRTRHLKLCGPVYTVNTDNDMLPGLEALNNAPQGFV
LFLNNVNERSEALAGDIYVLEAVRRKLGGLVINGAIRDIDDINKLDFPVFSTEVTYVSAKTAVTPATKVPDTVHVGDIVI
EPQDWIFGDNDGLLLIKRKYLNAVLQGALYVEQREEELKKEILSGKSMASLTGLDNYLAGKGPLKFDV
>lcl|NZ_CP026520.1_prot_WP_042225937.1_3192 [locus_tag=PC41400_RS16170] [protein=thiamine pyrophosphate-binding protein] [protein_id=WP_042225937.1] [location=3543166..3544926] [gbkey=CDS]
MYIKEIVALWLKRMGIHTVFGIPGSYVLPMIDEISRKEIRFVLSQHEYGAALMADGFAKAGAAPACVITTTGIGATNALS
GIVNSFTDSQPVILLTGQVPISTYGKGALQECVGEGRTFDLEALMSKVTKYSKLVESADHLLDTLKEAEMHLWSGRKGPV
HICIPIDVQFTEIHFEGFEPFTFPKPTEIVDAGGEEVSRLGNLLKESQKPLLLLGAGANYSHNEEVVEQIADLGVPVATT
LRGKGIINENHPHALGCTGLHGKSAANYYLYREADLVIAVGVSMSEFTTQGWDSIFDKAPLVQVDIDPKQIGKNYEVRIG
IAADSETFLLALYGLLEDASWEKATAALRQSKHKFAEADREKDLTYVSAVQGKLHPVQVIETLQKLLPQGKTIWVTDPVS
WTETHLLSKGAGTYIEGLNGAAIGYAPPAAIGAKCAKPDSYVVSIFGDGGFRQTALEMATATSHEIPVLWLNLNNEKFES
INNAQKSYYKGNVVGTSYKPIDYVAFGESMQIDTIRVTTVEELQEAAKSFLHEERPLLLDIQIVGSVVSAPKFRQLARYK
TWKLTTSRKSSEDMKEIVSYFLKNRY
>lcl|NZ_CP026520.1_prot_WP_042225939.1_3193 [locus_tag=PC41400_RS16175] [protein=methyltransferase domain-containing protein] [protein_id=WP_042225939.1] [location=3544959..3545777] [gbkey=CDS]
MSGGITENRARLDSSAASVRSVFDARTVGSSHRRLAELLEPGQSVLDAGCGTGAITKGIAEAVGQEGRVVGIDNNDGLIS
YARETFGGQETLAFETADIHYLPYRNEFDIVTTARVLQWLADPRQALGQLIEAAKPGGRVIVLDYNHDKISWQPEPPPSM
KAFYAAFLKWRSDAGMRNGIADELPGVFADLGLVEIQSSEQHEIVMREDGNFLQQAGIWADVAATRGVQMVADGYLSEQE
RAAAETEYRQWVQTEAVSQTMYMLAVEGVKPE
>lcl|NZ_CP026520.1_prot_WP_042225940.1_3194 [gene=cimA] [locus_tag=PC41400_RS16180] [protein=citramalate synthase] [protein_id=WP_042225940.1] [location=3545860..3547482] [gbkey=CDS]
MDNRITILDTTLRDGTQGEGVSLTVEDKLKIAKRLDMLGISFIEGGWPGSNHKDAEFFIRARELNLQQARITAFGSTRRK
NIPAEHDANLRQLLDIRVGTAAIFGKSWDFQVQTAIQTTLFENLNMIEDTVRFLKRSGIGVIYDAEHFFDGYKSNRAYAM
STIRKAYEAGADWVVLCDTNGGTLPHEIAAIVASVKEQLDCPLGIHAHNDCELAVANTIAAVMAGAAQVQGTINGYGERC
GNANLISVIPNLQLKLGLPVITAEQLAGLTSTARYVSEIANMHMPVNQPFAGHAAFAHKGGVHVSAILRDSSTYEHIAPG
LVGNRQRVLVSELSGQSNLVFKAQELGLELEANDPNTKILIERVKELEHQGYQFDGADASLELLIRSVHGNLEEILTLES
FKITVVKSVDQAILSEAVVKVKVHGESVYNAAEGNGPINALDNALRKCLMQHYPSLQTMHLSDYKVRVLDDKDGTATKVR
VLIEFSDREETWSTVGVSENIIEASWQALCDSIRYALLDVKPALNKRAEGAKEAVGLVNH
>lcl|NZ_CP026520.1_prot_WP_042225942.1_3195 [locus_tag=PC41400_RS16185] [protein=MFS transporter] [protein_id=WP_042225942.1] [location=3547518..3548777] [gbkey=CDS]
MSTKQPKKGETAHASGAGIQGPERTRMPLAVYFLAVSIFIICTAELLPMGLLPQIARDFNMNVSSAGLLITVYALGVAIG
GPLITAATGRIEKKKLLIGLLLLFVVGSLISAVTTNYYLLMVGRLFCSFSHGTLVGVAIVVVQRLAPPGKEGSAISYLTV
GLTVATILGVPVGTFIGNSYGWRMSFYALTIAAAICVIGMAWLVPTIKSGKAVGMAKQTREIFRAKPLVALLTTVFSSGG
LFAGYTYIAPYLEEITGFTPTAVSPIMLAFGIGSAIGNIASGKLADKGLMKTLLGSLAAMIITMTAFALLGQVPILAVIF
SFLWGLTGFAIVTPLNMLVLRKAGDAADLAASLNISAFNVGSAIGAAVGGVAINTALGLRGAPLSGVAIGLIGVLFAFTL
IWMDRKSAWQQKTVSKSST
>lcl|NZ_CP026520.1_prot_WP_042225944.1_3196 [locus_tag=PC41400_RS16190] [protein=FAD binding domain protein] [protein_id=WP_042225944.1] [location=3548829..3550250] [gbkey=CDS]
MNDFMKEQGKPRKAIVIGNSMAGKLAARVLSDYFDTVTIIEQDPQPEGPYPRKGTPQSYHTHGLLKSGENTLDKFFPGIM
QELEGHGGERLDFIRDISWVYQNFHFVRFNSGFEVVSQSRPLLEWRVHQRLLQRSNIEFAYSHRVVQYISSPGENRFTGV
ETENLVTRQSERLEADLIVDAAGSATKTPDQLLKHGFDSLSQSEVHINLAYATRMYERDPSVSHDYKSVIVFPNPPEVSK
GGLTFPIEGDRWSVTLVGYGDGNHPPRDDQGFLNFAKSLTHPHLYDHIKSLKPAGDIHVYRFPSQVRRHYEKLDRFPSGL
IVIGDAFCRFNPVFGQGVSIACLEALELQKVLQEVRTGKIKAEGIPNAFHKRVSALVEPTWMTATSDAFLYPNTKGKKPG
GLGFMQWYVRRLYRLSEYDQDVYGKFIRILHMVEKPTELFKPAMIYKVMRAKMPKPDKRTPGSPVNRDREQTL
>lcl|NZ_CP026520.1_prot_WP_042225946.1_3197 [locus_tag=PC41400_RS16195] [protein=Dabb family protein] [protein_id=WP_042225946.1] [location=3550308..3550610] [gbkey=CDS]
MSTIRHLVLFNLKHEEGSAEAEKFLSDGRELLASIPQVQKLQVLRQISLENDYQYAFSMEFSSQSDYDEYAAVPRHIEFV
EQRWLPEVVRSFEADFEVHS
>lcl|NZ_CP026520.1_prot_WP_042225949.1_3198 [locus_tag=PC41400_RS16200] [protein=hypothetical protein] [protein_id=WP_042225949.1] [location=3550878..3551642] [gbkey=CDS]
MKDGVGIGRKQIWRFIFSKLAATGTLFTVFVIIIIQNDAGLYQISEASRLPYLWWLVYGYAVIYSTIVELILFKFFPKAK
VGVRMIVYGAGGFAPFAVLIFDPIYLAISGLIGSVCALFFYAADCLYSRTKIMAAVAALCIPAVLLVFAWGDFTETKQWR
EVRTDSSYEVSFAYFNGEKAVPVPIKKGETALLTYSWTMKNGGYGVRFTDVLQVPGDLPANNKIRIRVPESGIASLVLTG
DHAEGTFRMEWKTE
>lcl|NZ_CP026520.1_prot_WP_042225952.1_3199 [locus_tag=PC41400_RS16205] [protein=flavin reductase family protein] [protein_id=WP_042225952.1] [location=complement(3551664..3552272)] [gbkey=CDS]
MIKQEVQDTFRTIRPSILYFGTPVVLLTTLSEDGSTNISPLSSAWALGSRIILGIGNGGKGLENLSRTGECVINLPDSSL
WEQVEKLAPLTGKSPVPDNKRELYRFERDKFSAAGFTPLPSVCVKPLRIRECPLQIEASLVQVHPSDGFCIVETESRAVH
AEERIVLGEHYVNPAAWNPLIYSFRHYFGLDAEKGKSFKSET
>lcl|NZ_CP026520.1_prot_WP_042225953.1_3200 [locus_tag=PC41400_RS16210] [protein=helix-turn-helix domain-containing protein] [protein_id=WP_042225953.1] [location=complement(3552351..3553043)] [gbkey=CDS]
MKASPDTITLAGLIGEPSRMHMLLELLSGKCMPASELAGAARITPQTASSHLAKLVEGGLLTVELSGRHRYYKLAGPEVA
DALEALNTLARPKPVKSLREYDRLNHLRFARTCYDHLAGEAGVALADRLLETGIIGQSGRDFTVTEKGEIFLRAFGINCA
ELRRSRRQFAKCCLDWSERRYHIAGSLGAALTEQLFRREWIERIPGGRAVRITDTGRDGLRHLGVDLNLL
>lcl|NZ_CP026520.1_prot_WP_042225954.1_3201 [locus_tag=PC41400_RS16215] [protein=GNAT family N-acetyltransferase] [protein_id=WP_042225954.1] [location=complement(3553426..3553920)] [gbkey=CDS]
MSTSADKSTILLNFYQNEYEDGLRSYYLPDEQLQFSALPHEVLKTAAQDAHRYPVVILADSKPVGFFVLYDGEEIALYSD
DPHTIVLRAFSVNHAEQGKSYAKQGLLLLPDFVRKHFPLARIVGLGVNMKNIPARNLYLKCGFEDHGRTKVGPLGPQHIL
TLPL
>lcl|NZ_CP026520.1_prot_WP_042225957.1_3202 [locus_tag=PC41400_RS16220] [protein=GerAB/ArcD/ProY family transporter] [protein_id=WP_042225957.1] [location=complement(3554040..3555143)] [gbkey=CDS]
MRSTSWQTFRFAMIYMISQPIAFLISPLITTSGYQGWISLFLGGLVSILLLCCTVYVGRLNPDVPWVQFGEQIIGKWPHR
LFLFLVLYWCVYYVSVDIESFTLFYGSTYMRETPQWFLQLIIGLVIVITARWGFVSIVYIADGLFLITMIGIAIIMGLFS
KNADFQMFPALWTHHKMGLAFRDTLTAVSWFGEWFVFLFMVPHLKFGAKTLRNLLSASLLVTFAVFMQWMLTLFTFGPYF
GSRIQYPLLELIRSSTLSGLLGNADPLLIGLWTTSMFVHDSFILYVGTKCAAELFKFKTRKAILPLLCGTAVVVALQFSR
DTTLFLFDMNSLSVVTFFVFVDALPVFYAVIAYVRIKFGKNFLQTSK
>lcl|NZ_CP026520.1_prot_WP_042225958.1_3203 [locus_tag=PC41400_RS16225] [protein=hypothetical protein] [protein_id=WP_042225958.1] [location=complement(3555176..3555367)] [gbkey=CDS]
MIALAFFALGLLEWIRTNPKPPGEAVKFWVLFSVLGCWSVLASAFTWWPQPNQLTHLLLGWLE
>lcl|NZ_CP026520.1_prot_WP_042225959.1_3204 [locus_tag=PC41400_RS16230] [protein=Ger(x)C family spore germination protein] [protein_id=WP_042225959.1] [location=complement(3555412..3556560)] [gbkey=CDS]
MRRIRLFIALLPLLLMTGCWSKVEIDEQTFVFALYIDQGDTPDKVEVTISSPLPNRLLSGQQAGSGTSHGQPYAVISKQG
TSIQQALMLIQRDLTRKIHFGHTRIVTVGRQYAENGIKDMLDWMQREPTFHISTIVSTAPGKAKEVAELTPVYERMPAEV
MLKMAQQRAMLDTSAKQCLIAEASGQGVATNYLTMGVMKMPSENGEIKPWAGIQGVALYYDDKLTGVLPSAEGRAIAWAL
GRSGRQVYTLSWDDGQSRADILFNQLMSKKSVRMGPEGPRFTLSLKGDGDLIHKRDIMNRDEKEMNAILEQKLREKIEKQ
LKSAFSKTQKAKSDVLQLGMLLDWKYPKFWEEKRDNWTDYYKNDLDIEFKVEINVKQVGILS
>lcl|NZ_CP026520.1_prot_WP_241688901.1_3205 [locus_tag=PC41400_RS16235] [protein=spore germination protein] [protein_id=WP_241688901.1] [location=complement(3556562..3558001)] [gbkey=CDS]
MKGLMYENIQQILAGSEDTVYQPFLLYERSCVLIYIQSIVDVKTLQEKVASALIAEAQGDHDWEEFLQLLDEGTLFPLPA
KLVYASQDAVREIVHGKAALCIEGLGCAYVFDVVKYQKRAVSESQNELVVIGPQEAFIEDIVSNLSLLRHKIKHPDLKTV
KYTVGTYTQTTVYMIYIEGLCKPEILKDLDKGIREISIDGVLGISYLSEQMKKGNKTPFPQFQYTERPDSVAASLLEGRV
GLIQDGTPSALLVPVTLLSLLQSSEDYYQSYLSASWIRLVRIFFALISMLLPSLYIAITTFQPEIIPSDLLLTISAAREN
IPFSALTEALVMELTFEGLREAGTRIPKPVGQTISIIGAIVIGQAAVQAGIVSAPMVIVVSITGIASYIIPHFELGLALR
LLRFPLLLIGGTTGLIGVFIASFIVYGHLTSQRSFGAPYLSPLAPLVLRDWKDIFMRVPSSLMKKRSKAYIDRNFRRQK
>lcl|NZ_CP026520.1_prot_WP_042225963.1_3206 [locus_tag=PC41400_RS16240] [protein=carboxymuconolactone decarboxylase family protein] [protein_id=WP_042225963.1] [location=complement(3558228..3558671)] [gbkey=CDS]
MKTRLQMDQLIPKGYAAMVGLEQYLGSTSLDKSLKELIKIRASQINGCAFCIDMHTKDARKHGETEQRIYALNAWRETPF
FSPEERAVLALTEAVTLVAETHVPDDVYDEVRLYFDEVQTSEIIMAIVVINAWNRLAVSARKMPDAD
>lcl|NZ_CP026520.1_prot_WP_042225965.1_3207 [locus_tag=PC41400_RS16245] [protein=RNA polymerase sigma-70 factor] [protein_id=WP_042225965.1] [location=complement(3558788..3559654)] [gbkey=CDS]
MTEALYKQYHSLLYSIAYRMLGTLNDAEDIIQDVYLEIQQQDLVRIGNMKAYLCKAVTNRCINFLQSARSRREIYTGEWL
PEPVSVQPGLDPAERIVEEETVSYALLVMMERLNPVERAVFILREALEYSYREIGDTLQKSEENCRKIYSRVKGKLLRKE
AKPLQKSEVEEQFVKLFLDAAKTGDFADFIAQMTADAVLVTDGGGKVRCALRPIYGSQRIRALFEGIWPRGYFSGSLLPA
RINGEQGFVLVRDERPVLAVSMTWDQEQSRVDRMFIVSNPDKLHGIRL
>lcl|NZ_CP026520.1_prot_WP_042225968.1_3208 [locus_tag=PC41400_RS16250] [protein=heavy metal translocating P-type ATPase] [protein_id=WP_042225968.1] [location=3559896..3562049] [gbkey=CDS]
MEEHRVKGLSCGNCTREMEEQIQKLEHGADARLSYNSGKLTVDSRISMPAVEKILGSDGAYLVKSQPAGGHTHEGHSHHG
HSHDHGHDHGHDHDHGGQSNRAITRNLVISAVLFAIAFFTDGRVPAYVSIPLYLIIMALSGYMTFIKGAKNLFRLKFNID
TLMTIALVGAIVIGEWKEATLVAILFGLNELLEGMGMEKARRSMETLLQVAPKEAVLLTDGQETVVPIASLKIGDLVQVR
SGQKIPSDGIVEEGKSSVNEAAITGESLPVEKAPGEKVFGGSINNEGLLKIRIEKAYQDSSLAKILHLVEEAQETKTPTE
LFINRFAKYYTPLIMIISLLVIVIPPLFLGGSWSTWLYQGLAVLIVGCPCALILSSPIAIVSGITRNARHGILIKGGVFL
EQLGKIDTLAFDKTGTLTKGEPHVEKSVVLDERKFYPAAAAIEKSSSHPLAKAIMKEIEAKGIRAEEADSIETVSGRGVV
ASFKGERYWLGNEKSMEHLRIPADIQEQIDSLKEEGLTLVIVADQEKVLGMFGISDEIREESREVIAALHKAGIHKTVML
TGDHRKTAEKVAKAVGVSEYYGNLLPEQKVDKIKELAAEGTVAMIGDGINDAPALASAQLGIAMGKGTDSAVETADIVLM
QDHLGKLPEAISVARQVNRVIKINISLALGLKAIALLLTIPGLLTLWIAILSDMGATIFVTLISLTVLIEKKPSSLS
>lcl|NZ_CP026520.1_prot_WP_042225970.1_3209 [locus_tag=PC41400_RS16255] [protein=metalloregulator ArsR/SmtB family transcription factor] [protein_id=WP_042225970.1] [location=3562177..3562542] [gbkey=CDS]
MQKPIEDLCDETCAGTETDGTALEHAKMDESMAVNLAEMFKALGDPTRVKIIYALSQQELCVHDLSVVLEMGQSAVSHQL
RYLRNVRIVKRRKSGKTVFYSLDDDHVKEIFRQTLQHLAHV
>lcl|NZ_CP026520.1_prot_WP_042225972.1_3210 [locus_tag=PC41400_RS16260] [protein=sensor histidine kinase] [protein_id=WP_042225972.1] [location=complement(3562556..3563836)] [gbkey=CDS]
MKDILVHLFIMTFPPHIYQIFMLTRFYKREKKRNRFLMGILMGFSGLFSISYPVMYLVPDFYWDLRGVPLLLAFIYGGYY
GGFTALGMMLIFHGIMFPSFGVSDVILYAAGFLPVLFRAREFYQYPSRKKIALSLIYTSMGTAALLIWLSYIVTHHGFVW
KDLLPQVSLLVTAQLLLPVLLILWFQHMLRHDQMSIEIQKAEKLQHLSDLAASVAHEIRNPLQVTRGYLQISQKEADEKL
GRYLSIAIGELDQAEKIISEFLLFAKPELTETGLINLGDSLQQAVTMLEPYASMSGAVIELSCEDHIFIAGDSSKLKQAF
VNLIKNGIESTQTDGLIRIRGYVRKHKAIITITDNGSGMTKEEIERLGAPFYTTKSKGTGLGLMVTTRLIDAMRGKMRFS
SEVGTGSEVMITFPLTTKNRVSILEE
>lcl|NZ_CP026520.1_prot_WP_042225974.1_3211 [locus_tag=PC41400_RS16265] [protein=arylamine N-acetyltransferase] [protein_id=WP_042225974.1] [location=3564101..3564886] [gbkey=CDS]
MRTDREDLKPEVEAYLNRIGYEGPLDGSYEALALLQEQHLHTVPYENLDILNGVPLSLDIKRLRHKIVERRRGGYCFELN
ALFGWLLRQLGYPVTDYVARFWRDETELPPKRRHHVLKVEAEGVSYLCDVGVGGIVPRRPVRLAEKLEQPQGEELYAMER
DPLFGWVLSEKHHGSWRRIYSFTEEPQLPKDFVMASFWCEHSPESIFKKDAMVAIRTREGRNSVSGKEFKLFTPEGVRTF
VPENDAAYKEALLKYFGIPSE
>lcl|NZ_CP026520.1_prot_WP_042225977.1_3212 [gene=metE] [locus_tag=PC41400_RS16270] [protein=5-methyltetrahydropteroyltriglutamate--homocysteine S-methyltransferase] [protein_id=WP_042225977.1] [location=3565315..3567612] [gbkey=CDS]
MNALTSSSLGYPRIGAGREWKKALESFWAGKIGEEELTDRLAEVRLGNLRIQREKGIELIAVNDFSGYDHVLDTSAMFGI
VPERFQYEGGEVSPGTYYAMARGEEGEAACEMTKWFNTNYHYIVPELQGAKPKLVENKPLKAYREAKETLGIQGKPVLLG
LFTFLKLSKGYAETETDAWIDRLLPLYGQVLRELEKEGAEWVQVDEPVLSTNVSAEDMNRIEAIYKTLTEAAPGLKIMLQ
TYFESVEHYDRVVKLPVSGIGLDFVHGGEKTLKLLKSGGFPEGKILGAGIVDGRNIWRSDLKEKLDTIRDILAVVPGERV
IVQTSCSLLHVPVSAVHEDKLDPVLRDALAFADEKLEEIVLLSKAAVSQTAETAEALQRNEKALAALRSSEARGNKAVQA
AALQLQKVPAARGEGFGERRALQREKWKLPFLPTTTIGSFPQTAEVRQARSKWRKAEWTDEQYKGFIRSEIDKWIKLQEE
LDLDVLVHGEFERTDMVEFFGEKLDGFAFTRNGWVQSYGSRCVKPPVIYGDVAFTQPMTVEETVYAQSLTTRPVKGMLTG
PVTILNWSFVRDDLSRDVVAYQISLALEKEVQALEEAGIEMIQIDEPAVREGLPLKKDEHGAYLDWAVNAFRSTTANVRA
TTQIHTHMCYCEFHDMIGSIRDMDADVISIETSRSHGELIESFEQHTYEKGIGLGVYDIHSPRVPSVEEMVSMIERALRV
LDPQLFWINPDCGLKTRGSEETVAALRNMIAAAKQCRETMAVSQA
>lcl|NZ_CP026520.1_prot_WP_009673767.1_3213 [locus_tag=PC41400_RS16275] [protein=YolD-like family protein] [protein_id=WP_009673767.1] [location=3567831..3568073] [gbkey=CDS]
MAKAKVKRPTRDEFELEELGNRLVEAKDEDSLILLTVWGWEEQVYGRITVMDPRTRLVHVQDGEQLTKVPFMDIMQVDNA
>lcl|NZ_CP026520.1_prot_WP_042225979.1_3214 [locus_tag=PC41400_RS16280] [protein=FAD-dependent oxidoreductase] [protein_id=WP_042225979.1] [location=3568181..3569728] [gbkey=CDS]
MNETNSKPAAAFPQFPESYWLASAQIPAYSRLNENIRVDAAIVGAGITGLTTAYLLAMKGLKVAVLNAGRILHGTTGYTT
AKITAQHDLIYDQYITDFGEEKARLYYEANRDALQFVRHLVKEHHIQCQFEEQDAFVYTDTDSSLHKVFKEYEAYEKLRI
PGSYVEKTDLPFETKGAVVMKNQAQYNPVPFLIHLAEEIVRKGGRIYEETTVTGYEPGSPSVLKTSGGFTVSSSHVVSAS
HFPFNNLHGLYFVRLYAERSYVLAVKTDKRYLGGMYISADKPKRSLRSVLINGEPMVIVGGEGHKTGQGICTFQYYEALE
KFARETFGVKEIAYRWSAQDLYTLDTIPYIGQELDDVPNLYVATGYKKWGMSTGIAAAQLNAKLILGEDSPYRDLFSPSR
FHANPSIKTFVKENANVAKELISGKLEFPTREAGELREDEGAVVQVRGKRAGAYRDREGSLHVVDTTCTHLGCEVDWNES
ERTWDCPCHGSRFAYTGEVIEGPALKPLAKVELDE
>lcl|NZ_CP026520.1_prot_WP_042225983.1_3215 [locus_tag=PC41400_RS16285] [protein=GNAT family N-acetyltransferase] [protein_id=WP_042225983.1] [location=complement(3569819..3571039)] [gbkey=CDS]
MSRRKLQERLKMKNVEPQHGEQFNHLLRYVFQVTNHDLQTFGWEDREIAQAKLPVLKQADVLGWFDEDKLISQLAVYPFQ
VNIFGRIYDMGGLTGVGTYPEYANMGLMNQLMRQALTNMRGRRQSISYLYPYSIPYYRRKGWEIISDKISFEVKDTQLPK
LKTVPGNVERVTIEHPDIQTVYQRFALQNHAAMLRNNLAWEEYLRWDLDDMTGAIYYDADHEPSGFLLYWIAGEIFHIKE
MVFLDEEARTGLWNFISAHFSMVTYVKGHTYKDEPMSFLLEDGDIKETIAPYFMARIVDVSQFVEQYPFQAQGEDCKLTF
NLHDPMLEWNSGTFTLKVDKEGQGRLADGGGTPSASFDIQTLTTMLMSYKRPSYLQRIRRFHADRESVQLLEKLIANEQP
YFSDYF
>lcl|NZ_CP026520.1_prot_WP_042225985.1_3216 [locus_tag=PC41400_RS16290] [protein=hypothetical protein] [protein_id=WP_042225985.1] [location=complement(3571179..3571436)] [gbkey=CDS]
MAGSLSDDLFIGSIISLLIWGCTSGIFTRFNLTHTEDRYIQPPDAEEYFSPKPDKKPFLHAFRSVFFIAAVLGFAASVVV
SYLEF
>lcl|NZ_CP026520.1_prot_WP_042225987.1_3217 [locus_tag=PC41400_RS16295] [protein=FAD-dependent oxidoreductase] [protein_id=WP_042225987.1] [location=complement(3571645..3573219)] [gbkey=CDS]
MTPIQLADVVIIGGGTGGFAAALASAESGKSVILTEETVWIGGQLTSQAVPPDEHPWIEKFGCTRTYRQFREGVRNYYRD
YFPMKPEAKAITELNPGSGSVSRLCHEPRTALAVLQQMLAPYVHSGQVTILLRHIPVSVETDGDAITSVEVKSLDSGALT
ILKAPYFLDATECGDLLPLAKAEYVTGAESYAETKEPNAVVGGPLPGDMQGFTYCFAVDHLEGENHVIEKPEMYDFWKAY
KADFWPDRLLSLMGVKPATNEPVKYEIFPGTEHFSLFRYRQIADIGHFREGTYRSNVSLVNWPQNDYWLGSIIDVPEEEK
QKHLYEAKQLSLSLLYWLQTEAPRPDGGTGYPGLRLRKDVVGTEDGLAMYPYIRESRRIKAEFTVLEQHVATACRPDGRA
EEFADSVGIGCYRIDLHPSTENRPYIDVSSLPFQIPLGSLIPVRVRNLLAAGKNIGVTHITNGCYRLHPVEWNIGEAAGY
CAAYCIDNGMTPSELRGNPDELAKFQSHLAKAGIELEWPALRPV
>lcl|NZ_CP026520.1_prot_WP_042225990.1_3218 [locus_tag=PC41400_RS16300] [protein=sugar phosphate isomerase/epimerase family protein] [protein_id=WP_042225990.1] [location=complement(3573225..3574058)] [gbkey=CDS]
MKLSLSMWSMHRTTREKQGTVLDFLAFCRQEGIDRVELLDVFWKDKEAEIPAVLSFLKQNGITLSSYAVGNDFVKGTPGE
RAEALRVVIDGIETAGRLGASVVRVFSGNMSDAYTFDSALEWIVSGLGEAAALAEQAGITLCLENHGKLAGKGDQVLEIL
KRVGSPALKSTFDTGNFLLVDEKPLHALDVLLPHIGHVHIKDFAETPAGRYEALNGKTYEGVFAGEGDARLDIIVNRLNE
AGYNGAYVLEYEGTGDETEGIRKSYQNFNHLVQTKGD
>lcl|NZ_CP026520.1_prot_WP_129112568.1_3219 [locus_tag=PC41400_RS16305] [protein=YjcZ family sporulation protein] [protein_id=WP_129112568.1] [location=3574306..3574392] [gbkey=CDS]
MGYGIGTSTAVILVLFILLVIVSRTFLY
>lcl|NZ_CP026520.1_prot_WP_042225991.1_3220 [locus_tag=PC41400_RS16310] [protein=catalase] [protein_id=WP_042225991.1] [location=3574596..3576725] [gbkey=CDS]
MKKDDQREPINEQSKDKQLETYRVDDTGQKLTTNQGLRVSEDEHSLKAGVRGPTLMEDFHFREKMTHFDHERIPERIVHA
RGFGAHGYFQVYEPMTEYTKAKFLQDPSVKTPVFVRFSTVAGSRGSSDTVRDVRGFATKFYTEEGNYDLVGNNMPVFFIQ
DAMKFPDFVHAVKPEPHNEIPQAQSAHDTFWDFVTSNTETAHMVMWAMSDRSLPRSFRMMEGFGVHTFRFVNEQGKGHFV
KFHWKPLLGVHSLVWDEVQKLAGKDPDFNRRDLWDAIETGNYPEFEFGVQMIKEEDEFMFDFDILDPTKIWPEEQVPVKI
IGKMTLNRNTDNFFAETEQIAFHPGHVVPGIDFTNDPLLQGRLFSYTDTQLSRLGGPNFHEIPINRSVAPVHNNQRDGMH
RMTINRGPVSYHKNSLAGNSPEPASEKEGGYAHYAEKVEGRKIQARSDSFKDHFSQATLFWNSMSAVEKQHIVNAFRFEL
GKVKTKEIRQRAVDMFSQVSVDLAKRMAAGIGAKAPENTSDSRVTLSSPALSQMNTVKTPLTRKVAVLAGNGFRGKEVSG
MLQSLKAVGLVPEVVSVQLGTIVCDGGNPLEVDHSLLTADSVLFDAVYIAGGRQSADELLADSKTADFVKEAFQHYKTIG
AVGEGVELLAAAGIAGSAAGSPPATATPEIQPGVLAAERADEAVGFSRDFAAAVAQHRHWDRSLQSQAR
>lcl|NZ_CP026520.1_prot_WP_042225992.1_3221 [locus_tag=PC41400_RS16315] [protein=MBL fold metallo-hydrolase] [protein_id=WP_042225992.1] [location=3576905..3577639] [gbkey=CDS]
MKITREGDLYQLAFLPRMFPVNCYLVEEKDGLTLIDAAMPFSAKGIMKAAGGIGKPIKRIVLTHAHEDHVGALDELKRLL
PQAEVFISARDARLLKGDATLDPDEPQTPVRGAVPKKVTTVPDVLLKEGDRIGSLLALSTPGHTPGSMSFLDVRSNAVIA
GDALQLRAGIAVAGQVKLLFPFPALATWNKSEALDSVKKLRDYSPSLLAVGHGTMLSNPAAQLDLAIAAAERNLAASLNG
RSSH
>lcl|NZ_CP026520.1_prot_WP_042225993.1_3222 [locus_tag=PC41400_RS16320] [protein=TetR/AcrR family transcriptional regulator] [protein_id=WP_042225993.1] [location=3577641..3578243] [gbkey=CDS]
MSPRIGLDEQTLLRAAAELADEHGFEAVTLASLAQKLQVRSPSLYNHIEGLGALRAKLALHSLSILHGDLAAAVEGLAGE
EAVFALGEAYAAFARRHPGLYEATLRAPDRNDSELERISAEIVGLIVGVLGYYGLEGDAAIHATRGLRSILHGFSSIERS
GGFGIALSVEVSQKLLISAFLAGIHVMKQEETGRPSPRSQ
>lcl|NZ_CP026520.1_prot_WP_042225994.1_3223 [locus_tag=PC41400_RS16325] [protein=glycosyl hydrolase family 18 protein] [protein_id=WP_042225994.1] [location=3578547..3580973] [gbkey=CDS]
MHATQAYGKKLLPLLLLVTLLLSLLPIYPSTSSGAAAQWAPNTAYKAKDLVSYNGVTYECLQPHTSLNGWEPPNVASLWK
KAEDPTPTPSPSPSPNPSPSPSPSPNPSPSPSPNPSPTPSPSPGTGAPAWQAGTAYKVGDLVSYNGKTYKALAAHTALVG
WEPATTPTLWKLDSSDPTKPEPTTPFCAPDWSASKIYAKGKIVGYKGKAYRAISDVHGIAPDDTTYNMWEVAGKISNDLC
PVSIPNNIDYGTANPGTGNTNAGKINPTGAISAGHPVSGTGTGGTKGGIDPATDPGGNHPGFDADTGGRVAKLPSGTLPL
QHNTYEMSAGQEVVEYLGDWAIYGRRFDFSKLPVKNVNRLVYGFAGVCFPAAQNTQDPGFPTTAPASVNRTCKQSKLPDG
AMAIADFEAAFLRTLPGQTGGKITGTESMYDLDPNDVAGVFGALYQLRKTNPNLKLDLSVGGWTLSEGFPWMASDTTRRK
AFVDSVVRFLEQFGFDGVDIDWEYPASDGAVIGMARKDDPENYIKLLKELRAGLDWLTKKTGKEYRLSSAIPATEEKLAK
IDWAEASKYLNRLYAMTYDLSGAWERELGHHTSLYPNPNAKDPATGKHVDLSLDTTVKLLKGYGVPANKIMIGIANYHRS
KAMKNGDITEYTNGLIGKSTFGDLAWKAKSLILGIAGVGSWEAGVVEGYDLYQNYLDKDLKPKNGYKLYTDKAANADYLV
NPIGSFITVETPRTVALKTQYAKDNGLAGIFGWQLEQDNGYNLNALNHILGNKLVSDVSDGKPKDQIAVCGQNVTAAECD
QLNASLKK
>lcl|NZ_CP026520.1_prot_WP_042225996.1_3224 [locus_tag=PC41400_RS16330] [protein=ATP-binding cassette domain-containing protein] [protein_id=WP_042225996.1] [location=3581217..3581948] [gbkey=CDS]
MLTIRNLTKRLTTGQEILSKISLQIDEGELIGLVGASGSGKTTLLKCIALKEKWDDGQFIYNDRDITSLGIAEKFKLKRD
WAFIEESPALDRKQTAVKNVLKGRFYEKSVMRVLTGTNARDEHVLAMDYLEKVGLLDKAEKKVEDLSGGERQRVSIARAL
VQGAKIVIADEPVSGLDPEAANSVLKDLRELCKREKVTVVCSLHNVEFAERYASRIWGLAGGKIVLDIAGRKLSSMERAK
IFD
>lcl|NZ_CP026520.1_prot_WP_042225998.1_3225 [gene=ilvD] [locus_tag=PC41400_RS16335] [protein=dihydroxy-acid dehydratase] [protein_id=WP_042225998.1] [location=3582063..3583748] [gbkey=CDS]
MAKQGMRSDMIKKGFDRAPHRSLLRAAGVKEEDFGKPFIAVCNSYIDIVPGHVHLQEFGKIVKDAIREAGGVPFEFNTIG
VDDGIAMGHIGMRYSLPSREIIADSLETVVSAHWFDGMVCIPNCDKITPGMMMGALRVNIPTVFVSGGPMKAGKDSKGRS
ISLTSVFEGVGAYQVGKIDDKALLELEQYGCPTCGSCSGMFTANSMNCLCEALGLAMPGNGTILAVSPERREFVKQSAKQ
LMELIKLDLKPRDIVTKEAIDNAFALDMAMGGSTNTVLHTLALAHEAGFEYPIERINEVANRVPHLSKVAPASDWHIEDI
HEAGGVSAVINELLKKPDAIHGDCITVTGKTLAENVAGAEILNKEVIHTLDNPHSERGGLAVLFGNLAPNGSIIKVGAVD
PSVGGRHVGPAICFDSQDDALYGIANGKVKEGHVVVIRYEGPKGGPGMPEMLAPTSQIVGMGLGAKVGLITDGRFSGASR
GISIGHISPEAAEGGPIAFVQDGDIIELDLIERKIELKISDEEMEERRKGWKGFEPKVKTGYLARYSKLVTNASTGGVMK
I
>lcl|NZ_CP026520.1_prot_WP_241688902.1_3226 [locus_tag=PC41400_RS16340] [protein=sigma-70 family RNA polymerase sigma factor] [protein_id=WP_241688902.1] [location=3584053..3584526] [gbkey=CDS]
MLLRGMGAEPQDAQDLTQEAFIKAYRKLDQHDPDKSFAAWLYTIAANIFKDSKKRKKIPTELLAEQDQIAEKGPEEILLL
SEKQKEMQYRISKLPVNYRLVLLLRYTNDLNYAEISEIMGCHVSKVQNDLYRARLKLKKMMIGEERIDHEMLEARRF
>lcl|NZ_CP026520.1_prot_WP_053228604.1_3227 [locus_tag=PC41400_RS16345] [protein=DUF4179 domain-containing protein] [protein_id=WP_053228604.1] [location=3584498..3586132] [gbkey=CDS]
MKCWKPEDFEQAVLKGADFSSVNMQDHLADCVGCRNRYTEWIEEQENWMRQLYPNVLPAAFTEGVMLSISSEEELSEDSV
ITAAETAAKEPKGIKKRWKLAAAAAALVLFMGAATFFSAPTLADIVRSLFAGYNADTGLLKAQKLGLVQHPNIEVKDKGY
MLKIDEAVADSTRVVVALQLFGPDGKHDRNKIAFYNKNHIFIKDEKGNKVGEMYDYGSTDDFYYMMAIFPEPVQTDRLTV
EGTIQQLSNTPAKTSITPGNWGFTFNLDLKEANKLTHITPLQGEYTGPDGMTVRLKRLTRMVQGVRLEFDTEISGEALKR
SPGELWKEQKLFFHFEDLAGNEIHSVNTRKMPAVDSVMIDGNTPGNKPGLMHWSYTFKELPIDSPYKFVLDGYFVAETDG
TSVEFNPQRLKEEPIKLYWKGDRLVLNDFMTVNQPTSEGDKLESSLYVTGDLHNEIFFDKWVVRMADKSEFTTQGRGILS
PEGDNWKDGVTVLSSINESNKPFEFRFPALTPVPDKLTLIRTVVGKQYTNTDWSLELFEKKNGR
>lcl|NZ_CP026520.1_prot_WP_042226004.1_3228 [locus_tag=PC41400_RS16350] [protein=acetamidase/formamidase family protein] [protein_id=WP_042226004.1] [location=3586330..3587241] [gbkey=CDS]
MAVHHFQPKHYHSTMGTHEPVLKIADGDSVVTSTVDARGWDYRGESVSGRGNPMTGPFYVEGAQPGDTLAVRLESMVPSR
DWGWTFNVLARNVVDPIAAAALPPFDIVRWSVDAEQALVVLEKPTSGLAGLKLEARPFLGCFGVAPPRDQAISTATSGVY
GGNMDYKGLVAGVTVYFPVFAEGALFYLGDGHAAQGCGEIVGTGVEISLDVRFTVQVLKGKTIGWPRGESDTHWFCIGNA
RPLDQALQHATTEMLAMLTKDYGLTASEASLLLGQAVEYEVANVFNPAYSVVCKVPKAVLSML
>lcl|NZ_CP026520.1_prot_WP_042226006.1_3229 [locus_tag=PC41400_RS16355] [protein=AraC family transcriptional regulator] [protein_id=WP_042226006.1] [location=complement(3587357..3588226)] [gbkey=CDS]
MNGLEKMNGALAYIEANLTNDIDFKEVARLALCSEYHFKRMFSTLAGVTLSEYIRRRRLTLAAFELASTDLRLLDVAVKY
GYSSSDSFARAFLSLHGINPSEARTLGHSLKAYPRMTFQLTIQGGSEMNYRLVEKDAFRIVGLKKRVPLIYNGVNPEMAA
MWESLNMETITLLKSLSNVEPSGLISASVNFSEGRLEYGELDHYIGAATTADCPDHLTQLEVKASTWAVFEAVGPFPQTL
QNIWGRIYAEWFPASGYESTEGPELLWNESKDTSSPTYKSEIWIPVKKK
>lcl|NZ_CP026520.1_prot_WP_042226008.1_3230 [locus_tag=PC41400_RS16360] [protein=TetR/AcrR family transcriptional regulator] [protein_id=WP_042226008.1] [location=complement(3588361..3588966)] [gbkey=CDS]
MPRNPERDQEIRKERCDQILSAAVALFAKNGLSSTKISDIAHKAHMSHGLIYNYFQSKEEIYISIVEKNLCILTSLFEQA
HRLPDASSYEKLTWLLDQVYCERWDDAVFYQVVADQILTSDSVSDQLKTSVRTSIGDNLKLLTRIFRVGQEKGELIQGDP
RELAFYFMNMIQAVLLAETRGIYFTGTPLHRNLLQLFLPKA
>lcl|NZ_CP026520.1_prot_WP_042226011.1_3231 [locus_tag=PC41400_RS16365] [protein=tetracycline resistance MFS efflux pump] [protein_id=WP_042226011.1] [location=3589170..3590384] [gbkey=CDS]
MSTMKNRRSPLVILMINMFIAMLGIGLIIPVLPEFLKEFGLGGKTAGYLVAAFGLTQFIFSPIGGEWSDKYGRRIMIVSG
LALFTVSNLLFAMASEVWMLYTSRLIGGIGAAAMIPSMMAYIADITTDEKRGKGMGMLGAAMSLGFVIGPGIGGFLAELG
LRAPFYVSSAVGALAMIGSLLVLPESLTKEELAANRSSQVKKESIFIQLGQSFKAPYFILLLLVFALTFGLANFEAVFPL
FVDNKYGYTAKDISILITVGALIGAVIQAMLIDKLIRRFGEKKLINLTFILSAVCLILMLLSGNFWYILVITMMFFTLTS
IMRPAINTLLSKMAGDEQGFVAGMNNAYMSLGNIFGPAIAGILFDVQINLPYMFGAVILVISFIISTSWSNKNKQKLVPA
GGSN
>lcl|NZ_CP026520.1_prot_WP_042226014.1_3232 [locus_tag=PC41400_RS16370] [protein=hypothetical protein] [protein_id=WP_042226014.1] [location=complement(3590446..3590886)] [gbkey=CDS]
MFGWMLFLHITGLLGWLGALLGILVTMLLLRKQLGTSDANSLAGKTIRVFGRIAHPSSILVLLSGVYLILQMGIGTDKPL
WLQVMEKGGGTIILLAVIFTSVLGSKMNKRLNSSGGTSVRLTGYLSTVGAFMFLIVSVVLVVSLKL
>lcl|NZ_CP026520.1_prot_WP_042226016.1_3233 [locus_tag=PC41400_RS16375] [protein=MMPL family transporter] [protein_id=WP_042226016.1] [location=complement(3590916..3593156)] [gbkey=CDS]
MSIKNNPWNKLARAVSSPRGAKIVAAVWIIAIVALALAAPGSKQVAVSSGEGSAFSQTSSEEAQKIIDERFPSEDGLTAL
LVFHGKNPITAEERAKIAGISEWLASAGKPPQIAGALPFHTLPKAVQDKMFSDDGTTILLNAAFQKNLESGQLYEGLQQI
RTHVEQMGIGSLDFKITGPAGIAADTTSLFKNADFVLMFATVGLILVILMIIYRSPLLAVIPLVISGMVYMVVDRILGLA
GKYGWFLVDKQALSIMMILLFAVLTDYCLFVLTRYREELQTSADKYEAMRHALSRVAEPILFSGGTVLIAMLTLFAAVFK
PYHHFAPVFSAAMAVIILGGLTLIPAVFALLGRKAFWPFVPKLSKSSAGKTRGFWASLGGFVTRKPGLTAGILLLALLAS
SLSVTSMKYSFNLLKSFPDSTSSRQGFEILETKFPPGQLAPVSVLLTSEKRLPLDAGFYEKVAALTGLLKNQPGVNAVSP
EIMPALPASGSQLPKNVLAADGTALKLQLTLESNPYEPVSLDRIDLLRGSSPALLEKSGLSPSEFTLHYAGQTAEQLDVR
NMNKRDTILLFSLITLFITIMLLIQTQSFKHALTMIATMLLSYTATLGLGWMIFHFVLGYDSISYRLPVYTFVFLIALGV
DYNIMLVSRIKEEASRYPWKEAVSRGVALTGGVISSAGLILAATFTVLITQPLQELFLFGLTMALGILLDTFLVRGMLLP
ALLVFTGDRKKSGITRETGTDLTAPR
>lcl|NZ_CP026520.1_prot_WP_042226019.1_3234 [locus_tag=PC41400_RS16380] [protein=HAMP domain-containing sensor histidine kinase] [protein_id=WP_042226019.1] [location=complement(3593222..3594601)] [gbkey=CDS]
MKTLYVRIVVTFLLIAVISSAAALLLSNSYYSAKLRDYNEQKILRISQEIRGLYEEIPGLAPGEYLTRIANMGFQVYAVD
DSLRGTFYGSPFKHKEIERDTIVRVLNGETYHGVMEERHLLLVTGFFENSIRNSIGLPVRSGGKTIALFIRPDMEQQIGE
VRILMALLLGFTFLLSLVLIVISTRFIVKPVQKLTEATNKIVGGDYNIRMDVTRRDEIGNLARHFTQMAQSLEKLDQMRQ
EFVANVSHEIQSPLTTIQGFAQSMLEKGASPEEERRYLRIIEEESRRLSSLSKELLTLAALDKETSALKPSSFRLDEQIR
QIFITTEWQWTEKELQLQLDLPELVIRADSQLLHQVWLNLIANSIKFTRPGDTISVQIRQEQQEILVSIGDTGPGIPPED
LPHIFDRFYKGDKSRNRSRSGSGLGLSIAHKIVALHRGSIEAKSLPGEGTLFQIRLPKL
>lcl|NZ_CP026520.1_prot_WP_042226020.1_3235 [locus_tag=PC41400_RS16385] [protein=response regulator transcription factor] [protein_id=WP_042226020.1] [location=complement(3594598..3595278)] [gbkey=CDS]
MKTHLLIADDDAHIRELLQFIFKKENYLVSEAADGEEAAALLEREQIHLAVIDVMMPGKNGLELCEEIRSYYDIPVILLT
AKGELEDKEKGYLSGTDDYLVKPFEPKELLFRVKALLRRYQLVSSEVIALNRTVIDRKSYEIHMNGETYVIPLKEFELLA
QLASQPGRIFTREQLLKLVWGADYEGGSRTIDVHIKRLRERFHEKTSDFVLTTVRGLGYKLEVCST
>lcl|NZ_CP026520.1_prot_WP_241688669.1_3236 [locus_tag=PC41400_RS16390] [protein=YceI family protein] [protein_id=WP_241688669.1] [location=3595564..3596289] [gbkey=CDS]
MNRSRYWWFGTLGAVLILGSLTGYIAIARDTLWVQEKEIQRGDKLGPLQLAKTWKVTAGSSAAVSVLSPHESLDLTFPKL
SGLLTINKEETEPVVGTIAADMTSLYSEDPAWNAKLRDWYFFNVKSYPEASFRIVKVGGFEKPLLEGEPHPLTIEGEMTV
RNMTQPIQFKGEITHTGGRMELKARCSGLETKKFGIPELLEPIYTGKNVATLEVTLILEPEDPDTSRHRQADSHNFREIG
A
>lcl|NZ_CP026520.1_prot_WP_042226269.1_3237 [locus_tag=PC41400_RS16395] [protein=methyltransferase domain-containing protein] [protein_id=WP_042226269.1] [location=3596357..3597175] [gbkey=CDS]
MKIDLGCGLRKHPGFIGIDKEPLPGVDLQCDLNDGIPLETGTVDFVMASRSLEYVPDLMNVMKEIYRVCRHKAVICLLAP
YAHTSYHALNPKFRSRFDEQLPRFFTNELYVYDPMRENSYMTDVPAGWDKEAKAEIDLRLLRCEFFYLPQYLSPIYDEED
RRMLRQSTLNVVDEILYYFVAVKEPISQQELDELAQGTYEEPVLLAERRREEAIREESRRPPESTFRHEPGSVLPQRDPL
PVPRKRRKGSKKPVKKKPAARKPAKGLRKRKQ
>lcl|NZ_CP026520.1_prot_WP_042226272.1_3238 [locus_tag=PC41400_RS16400] [protein=GNAT family protein] [protein_id=WP_042226272.1] [location=complement(3597299..3597853)] [gbkey=CDS]
MTITTDLIYFRYLTEEDAPALLSLMHRNRAHFEPFTITRPDTFYTLEEQTARIRMWDENRSADKSYNFGLFFKENDELIG
ELSLFHVAREPAHKCMMGYCLDHAYQGKGIMSQAARLALDFAFDEAGFHRVEAGVMPHNVGSIRVLEKAGFQREGVARKN
VKINGVWQDHVMFAILEEDERPRS
>lcl|NZ_CP026520.1_prot_WP_042226023.1_3239 [gene=xerS] [locus_tag=PC41400_RS16405] [protein=tyrosine recombinase XerS] [protein_id=WP_042226023.1] [location=complement(3598084..3599166)] [gbkey=CDS]
MNIQKQKDREELDKKIPSMPWYVDKFIHHKLPDLSPSTLLEYVRDYETFFSWLLAEGLSTAERMADIPLEDLEKLHMESV
DHYRAYLATKKDQANTKITINRKISSLRSLFHYLSQIAEDENYYPLLKRNVMAKIEIKRTNKAKDLAGKLEGKLLQENEI
FEFIDYIRSGYEHDIANNKQALYAYELNKIRDACIVSLILHSGLRVSEVVNLTQDDLDMKKRTVYVYRKGNNDGSFKTRV
FFRQEAVDDLTCYLQLRDTRYHPPKKEKALFLTIPNGKKEGQLMSKRAIQEMVIKYAKRFGKPFLSVHKLRHSFATDYYL
QNDLYKTQEQLGHASSDTTQIYAHLTDKTMEEAIDKRKEG
>lcl|NZ_CP026520.1_prot_WP_042226025.1_3240 [locus_tag=PC41400_RS16410] [protein=DUF421 domain-containing protein] [protein_id=WP_042226025.1] [location=3599493..3600173] [gbkey=CDS]
MDFMYIGIKLITGFIGLWGMTRLLGKKEISQLTPFDFVSALMLSELVGNTVYDKEVKYLELLFALAFWTVLSLTFEKITQ
KYKKLRRAFEGSPAVVIEKGKLNVKAMRSNSLDLEQLLMLLREQGIFSLHEVAYGIFENNGSLSILKKSPYDTVVRGDLN
LPDEPVELPVPLIEEGEIKFKELERIGRDEEWLKEQMKESGFKQIKDIFYAEWHREHGFYASGWGK
>lcl|NZ_CP026520.1_prot_WP_042226028.1_3241 [locus_tag=PC41400_RS16415] [protein=DNA-deoxyinosine glycosylase] [protein_id=WP_042226028.1] [location=3600192..3600704] [gbkey=CDS]
MLCMGFEPVIAPDSVVLILGSMPGELSLERQEYYGNPRNDFWRLLASLHGEDHVPGTYEERLRFALGRKIALWDVLASCE
REGSLDANIREPAVNDFDALFAEYPNLRTVLFNGAASEKLFRKHAGHLLADSRLGFVRLPSSSPANAMGWERKLAAWREA
WQGVTGRGEA
>lcl|NZ_CP026520.1_prot_WP_042226031.1_3242 [locus_tag=PC41400_RS16420] [protein=iron-containing alcohol dehydrogenase] [protein_id=WP_042226031.1] [location=complement(3600888..3602051)] [gbkey=CDS]
MKNFTFHNPTKLIFGKGQLQQLPKELTAYGKRVLLVYGGGSIKRSGLYDQVVSLLKEAGAEIFELAGVEPNPRVSTVKRG
ADLCKKEKIDVILAVGGGSVIDCTKAIAAGAKYDGDVWDFMTRKASPADALPFGTVLTLAATGSEMNSGSVITNWETQEK
YGWGSPLVFPKFSILDPVFTATVPKDQTVYGMVDIMSHVFEQYFHHTPNTLVQDGFCETILKSVIETAPKLVNDLENDEY
RETILYSGTMALNGILAMGVQGDWATHNIEHAVSAVYDIPHGGGLAILFPNWMTHVLDENVGRFKQFAVRVFGISADGKS
DRDVALEGIEALRSFWNSIGAPSRLADYDIDDSKLDVMVDRAMVYGPFGYFKRLDREDVTAIYKASL
>lcl|NZ_CP026520.1_prot_WP_042226034.1_3243 [locus_tag=PC41400_RS16425] [protein=organic hydroperoxide resistance protein] [protein_id=WP_042226034.1] [location=complement(3602158..3602574)] [gbkey=CDS]
MKTMYTTSATAQGGRKGRVATPDGKLDLQLSMPKEMGGDGGEGTNPEQLFASGYSACFLSAVELVAKKKGISTENAKLTA
EVSIGKNEDGSLQLSAKLQLSVPGADKSEVEKLLEEAHQVCPYSRATRGNIEVDLQAV
>lcl|NZ_CP026520.1_prot_WP_042226036.1_3244 [locus_tag=PC41400_RS16430] [protein=hypothetical protein] [protein_id=WP_042226036.1] [location=3602820..3603017] [gbkey=CDS]
MRLPENREIPAAGDLELLSMLETTYAVVTCRGSREEMMNLYGRLEEWIGGSGYEPDNEEGVTLLK
>lcl|NZ_CP026520.1_prot_WP_042226038.1_3245 [locus_tag=PC41400_RS16435] [protein=PH domain-containing protein] [protein_id=WP_042226038.1] [location=complement(3603118..3604644)] [gbkey=CDS]
MNEPRKLHKSYILFPVLKVLQTFVPIFMLLAIREGEWRWIPFLMLFGVGALFAGFGAVGWSTFRYRLEEDRLIIQKGLIF
HEEKMFYYPRIHSMNTEQPFVQRLLGVVQLKFEMPGGKNESDGVLPAVSAQEAARIIREINAYLSSSGYSPVAAENGEGD
GSTEKAVEPNAVPAVSKSRLPHTASAEKTGVYAYRATAGTLILAAFSTLNIGLIAACVAGVLSLADNVLPGNVYARLWNA
RNELHGGWLVYVLLGLGGLAAAWLLAGLLYLLKYAGFTVRAEAGTLTVSYGLLEKKKHTFQEARVQAVLIKEGLVRQLLG
RAQIELRVVSSSKGEKIMLMPLVSAKRIPQLLPQLLPRFSLPDIESRPPARALLYYSRTDGAVVAVLAAAAVYFWGPVAL
WGLLLLLLDFLWSYSRFRSSGMTLQGNQLTLRGRMIARSTWLVRRPQMLSLRMKGTRSQRRKQLLSLSVSVMGDTGSAAC
EVSFIEERDAQRVWSWFSRSADHAQAKD
>lcl|NZ_CP026520.1_prot_WP_042226039.1_3246 [locus_tag=PC41400_RS16440] [protein=PH domain-containing protein] [protein_id=WP_042226039.1] [location=complement(3604637..3605116)] [gbkey=CDS]
MNRPLTRRLDRRFVTVSQISALLTNGILLLASLVYLTVAQSRDWLMPPGWIAVAVFAVSLVLFTFVIPQARFRLYAFEVT
EEELEIRSGWIWITNILVPMVRVQHVELERGPLQRHYGIAELKIVTAATTHRIRGLSEDDAEALKHRLGVLARAVDEDE
>lcl|NZ_CP026520.1_prot_WP_042226041.1_3247 [locus_tag=PC41400_RS16445] [protein=Stk1 family PASTA domain-containing Ser/Thr kinase] [protein_id=WP_042226041.1] [location=3605375..3606676] [gbkey=CDS]
MEQRIGERYVLTEPIHPLADGVMYAGKDLSLHRDVFLLMVEKSGDASSSDYLRVLGQAAQFSDERFFHILNAGLAENHVY
TVLTAQKGTPLPAQLGRHRMRGIEIVSAVFDLGKGMQEAMEEGISGFSVTAENMWLTEDNRLKVMNYWSPGKLISRGSLG
LCTLLYQISTHSKQGPATFDQYQENVVPALTGLTSGQKEALLALSRRVFREEVSLSSFIFSLREIIALDKSPEAPPKVFV
PPAAPKEPVRETAPPRIQYRQEEEPAPERVRRTEKPPRAKEPEAEVADTRERMPLRRTIWTLGTAAIVLLGIAVFMYMLL
PGGPNKGPGSGGTAGNTATATPAPTPTPMSNTATPEPTPTPTPPNDAVPGQIPNLIGMTQEAAEQAVKAAGMRYSYFIEP
NPAESGTVFRQDPPPNQPANKGDRVTFYISKGQ
>lcl|NZ_CP026520.1_prot_WP_042226043.1_3248 [locus_tag=PC41400_RS16450] [protein=hypothetical protein] [protein_id=WP_042226043.1] [location=3606867..3607424] [gbkey=CDS]
MEKEWSMWGNERYLFDLLMHSEGSSIKMLEIILQEQLSEEVVEQVLIPSKSLHKKPRAVFGQTSPLLRRVSSYRYREETI
LHKAIYAHLEALPEGTLDGMNYEELQLEKLIDSLETKREILYAGYENNRSADVCFHPVKLKADLFPVKEYQLTCNGSVCF
LVHELFDAELILRHSRRKLRKEIFM
>lcl|NZ_CP026520.1_prot_WP_042226046.1_3249 [locus_tag=PC41400_RS16455] [protein=PQQ-dependent sugar dehydrogenase] [protein_id=WP_042226046.1] [location=3607597..3609021] [gbkey=CDS]
MIKVRASLRPLVSKINLPTVLKTTILPGDSIERLFIATQVGEIFYIGNGVIKTFLDIRSRIIKLGVSSGGYDERGLVGLA
FHPEFYYNGLFYLHYSVAGTQGPGALPGAFESFKPNPCDSKTLNLKWINREIQYDHIDTVEEWILQSNGQPQKRRTLLHI
RRPFLNHNGVNSLNFSPETGKLVLTTGDGGSGYDPFNLSQDDMEIAGKIIEIDVVKNTFIVNPPVVTRFNELPVPIQETL
TVIAKGVRNIPGISFQRFHDQYIKYAGNVGQDLVESIFSFIQYKPIPVTQLVQAFLMKSEPDQEGFINFGWRGWEGAFPT
AIITGCSANPALDEKTIAYFDEAVKTSVRRLQPLTNYFHKDPRPDKFGATALTGVQPYMGNEIPALTGNVVFTDLARAKE
SQLPVRGVLAYTTVRTDCHLNDFSIIETDYNFGSQSAYYVSLGANLNQSRLYLGVYGSMKVADFNQGTVFEIVP
>lcl|NZ_CP026520.1_prot_WP_084706485.1_3250 [locus_tag=PC41400_RS16460] [protein=YmaF family protein] [protein_id=WP_084706485.1] [location=complement(3609091..3609480)] [gbkey=CDS]
MKIPVTGFVMDSGDDDAHHSHKLFITSWNGTPVHVHAFSGITSIEDGHSHEYASWTAPAPTGVPHVHGYHTETSLNQGHI
HMIQGTTGPAIDLPGGGHYHLFEGYTTINGSHPHSHAYSGRTGNEVSRV
>lcl|NZ_CP026520.1_prot_WP_042226047.1_3251 [locus_tag=PC41400_RS16465] [protein=hypothetical protein] [protein_id=WP_042226047.1] [location=complement(3609574..3610146)] [gbkey=CDS]
MSRPVMNVALVCGPDCDMEVQAIRSTLEYLGARVFTYWIGRPNDLIDLLSGKDLYPNTDLILLCFHGDEGQFIMPELGES
SYDPGEPKGNFGPEEIRRFAKLAGKTVIGNGCSLGELETAQAFLDSGCEIYIGPNDYPDGNDALMFVLRLFYDLIQNKRS
VKEAFQNAKSLDAGMDMYQLYENAQQSSRK
>lcl|NZ_CP026520.1_prot_WP_042226050.1_3252 [locus_tag=PC41400_RS16470] [protein=hypothetical protein] [protein_id=WP_042226050.1] [location=complement(3610216..3610401)] [gbkey=CDS]
MRTEDQVQRKLTELNKQKQSVQERLNSDPENDFLKAQVEKLEDMTLMLEWVLNAPTGSYHS
>lcl|NZ_CP026520.1_prot_WP_042226052.1_3253 [gene=cls] [locus_tag=PC41400_RS16475] [protein=cardiolipin synthase] [protein_id=WP_042226052.1] [location=3610616..3612061] [gbkey=CDS]
MDILMKLYSFVPLLNFLFALAVVFMEKRNASVTWAWVMVLIFLPGIGFILYLIFGQNMSRRKIYKLKEFNQKDIDNLIES
QRRGFRESQVRYKDPVMADYQDLIYMNLTSSSAVYTQDNEVEIFTDGNDKFASLLASIKEAQSHIHLMYYIVKDDRLGRR
VIDALAEKAKEGVEVRFLFDDIGSSWLPRQFYKPLLDAGGEVAIFFPSRIPYLNFRVNYRNHRKLAIIDGMTGYIGGFNI
GDEYLGINERFGDWRDTHLKVTGSAVLQMQAQFFMDWNLASSHYLKAVSRYYPLIKHPGSVGIQIVSSGPDNELEQIRNG
YLKMIYAAKESIFIQTPYFVPDESLLTALKMAALSGVDIRIMIPSIPDHKMVYWATKSYLGELLKLGVKGYLYEKGFLHA
KTIVIDGKVASVGTANLDMRSFKLNFEVNAFLFDTRTATRQRLIFEEDMMYCSQFTPDMYKARSLVDRMQESVARLVSPI
L
>lcl|NZ_CP026520.1_prot_WP_042226053.1_3254 [locus_tag=PC41400_RS16480] [protein=YebC/PmpR family DNA-binding transcriptional regulator] [protein_id=WP_042226053.1] [location=3612282..3613001] [gbkey=CDS]
MGRKWNNIKEKKASKDASTSRIYAKFGREIYVAAKQGEPDPESNRALKVVLERAKTYMVPKAIIDRAIEKAKGGGEESYD
TLRYEGFGPNGSMVIVDALTNNVNRTASEVRSAFNKNGGSMGVSGSVTYMFDPTAVFGVQGKSSEEVLELLMEADLDVRD
IIEEEDTVIVYAESDQFHAVQEAFKNVGITEFTVAELTMLAQNDLELPEDAQAQFEKLVDVLEDLEDVQQVYHNVDLGE
>lcl|NZ_CP026520.1_prot_WP_042226055.1_3255 [locus_tag=PC41400_RS16485] [protein=GntR family transcriptional regulator] [protein_id=WP_042226055.1] [location=3613205..3613576] [gbkey=CDS]
MSFRIDDSRPIFMQIAEKIEDDIIEQRLPEETQVPSTNQFASFYQINPATAAKGVNLLVDQGILYKKRGIGMFVAEGARQ
GLMEKRKTQFYEQFVVSMVQEAEKLGITTEHLIEMIRKGDRNG
>lcl|NZ_CP026520.1_prot_WP_042226058.1_3256 [locus_tag=PC41400_RS16490] [protein=ABC transporter ATP-binding protein] [protein_id=WP_042226058.1] [location=3613573..3614472] [gbkey=CDS]
MKEVVVEVNGLTKTYNEVKAVDGISFSIEPDKIYGLLGRNGAGKTTVMQMVTAQMFPTSGEVKVFGQPPYENYDVMKQIC
FIKESQKYPDNFRVADVLKVAAAVFPNWDGDYAEALIRDFGLPLKRNLKKLSRGMLSSVGVVVGLASRSPLTIFDEPYLG
LDAVARSLFYDRLIEDYTRYPRTIILSTHLIDEVSRLLEHVIVIDSGKLILDAETEELRGMAYTVVGPAAGVESFVKGKE
VIHREPFGSMLSVTITGDSESRAKLLAADMGLEIAPVSLQQLIIYLTKTNLDEKAVKGS
>lcl|NZ_CP026520.1_prot_WP_042226062.1_3257 [locus_tag=PC41400_RS16495] [protein=hypothetical protein] [protein_id=WP_042226062.1] [location=3614469..3615158] [gbkey=CDS]
MNLQRLSGIMSIHYRNKVLWLYVFGGILLGNFLVNYIVSRLTGGVEIYTGGMSSIFVCLFIGGIFTAAQTFPFALGFSVR
RTDYYNGTLLMFTGVSVLTALLILLLSLLEVGTGNWGTNLHFFHLPYVTDGSILAQFMIYFSLLLYNLFAGFVIGSFQRK
FGGKGTFILMVVLFLILTILALLCTTNGWWGAIFNWIAGHTAVQLAIWLFPLTLIFAVASYFLLRRASA
>lcl|NZ_CP026520.1_prot_WP_042226065.1_3258 [locus_tag=PC41400_RS16500] [protein=mandelate racemase/muconate lactonizing enzyme family protein] [protein_id=WP_042226065.1] [location=complement(3615264..3616382)] [gbkey=CDS]
MLIHTIDTFPLFYRLPAAYGDANGIKSYRTTYLIRITTDTGIEGWGECVDWLPTLQKGFQERIIPYLTGKPVVERRKLVQ
TLKKWHSRSATAVSMALTEILAKSSGLSVCDLWGGKYRDKVPVYASFQSYSREPDWHKHSLEAIRGAVTDGFNKIKIKIG
GKSITEDQRHVLSVQNLLQGKVSLALDANQSYDVAAALQWKTLLEKWPNLMWLEEPLQIKYIAEYIALRDRLFVPLAGGE
NNKMASDFIPLLSQHALDFITPDPLHITGIDEFRETLSLARKFGIRAAPHAYDGALTRLYALFAQACLEPWSKMETEMIE
PVEWDAMDNPFNRLISVQPSNGEVAIPYGIGIGIEIDLDMLAFYQWDGSSYG
>lcl|NZ_CP026520.1_prot_WP_042226066.1_3259 [locus_tag=PC41400_RS16505] [protein=DUF2268 domain-containing protein] [protein_id=WP_042226066.1] [location=3616684..3617610] [gbkey=CDS]
MNIKTLRSDHIYQKVAQASPKEKLELFRNEMMAPFMKQWQIQQIPFRAEEANGFDVITMNNMMHRAPEQITPQISAEIEL
ISSETFWLECELAVSKSLQLFVEHEIKLPVSEYLFTIQLGNPESRALVINEGYSGFGGIPGFIWGTLLPNEYTIPRMKAS
LAHECNHNVRYQFIQWDHTVNLGELIVSEGLAENYATFMFGEELLGPWVTKTNADTLNRRIKPVLRDQLQLTGFDQFAPY
LYGDEIAKLQNFEPVNMPSSAGYACGYHLIQYYLRKTGKTIFEATITPAAQILDKVKGFWDEETIISG
>lcl|NZ_CP026520.1_prot_WP_042226068.1_3260 [locus_tag=PC41400_RS16510] [protein=MerR family transcriptional regulator] [protein_id=WP_042226068.1] [location=3617585..3618334] [gbkey=CDS]
MKKPLLAVKDIVQITGITSRTLHYYDRIDLFKPTHLTEKGYRLYDRSSLEKLQTILFLKELDFSLKEIADILKLTRQEQK
QLLKKQSQTLLSKKQRLETIMVALEEYVSGKDISHLEIFNSSSVVPLKEQYANEARFIYGETEAYKVFNETLDKLSPDEK
EERFQSMEEVFKQIASCVDPHPSSDEVQRLIEEWKKNLMQFMPCDTELLACIADTYKFDARFNNYFNQYGNENLADFLHS
AIMHHISQS
>lcl|NZ_CP026520.1_prot_WP_042226275.1_3261 [locus_tag=PC41400_RS16515] [protein=tyrosine-type recombinase/integrase] [protein_id=WP_042226275.1] [location=3618510..3619451] [gbkey=CDS]
MEGKRTLKDSDEIKERYAEQIRAFSLWMKHSGYTEATQKEYLREVLSFLQSLEGSPIEEAGKIGVINFLVSKQKTAGEST
RNRSLSAIRSFYSALIDFELATKNPALEVKKSKTEKNKKPVYLEEEELVQSLEYINGRYRNRNIAIFLLMGYCGLRVGEV
HRLNLVNFKKTKGAIEVFGKGRKWNEIPLPNVLNDYLIKVLEERLTPYSQKEDAFFVSQKGRRLSIRQIQKIANQTFSAF
KEKNPKVTDMNLSCHKLRHSFATMMLKKGNDIRIVKELMGHSSIETTALYTHVNDEQKKNAMASVNIPMEAIG
>lcl|NZ_CP026520.1_prot_WP_042226070.1_3262 [locus_tag=PC41400_RS16520] [protein=VOC family protein] [protein_id=WP_042226070.1] [location=complement(3619806..3620300)] [gbkey=CDS]
MNNKIQKITPNLWFDTQAEQAAELYTSIFKNSKIGRITHHGKDNHGIPEGTVMTVEFQLDGQEFVALNGGPYFKFTEAIS
FIVNCEDQDELDYFWEKLSEGGDEKAQNCGWLKDKYGVSWQIIPSTLTEMVSDPDPEKSERVMKALLQTKSKIDIKTLMQ
AYEG
>lcl|NZ_CP026520.1_prot_WP_042226072.1_3263 [locus_tag=PC41400_RS16525] [protein=hypothetical protein] [protein_id=WP_042226072.1] [location=3620900..3621910] [gbkey=CDS]
MVIKDRATWNKQTSECKEQMLRELNARLPAGFTFLKMERFERFGLGMETGIFMYEGCEFVYVPGDRVTLGWGHWAVGPDE
KTKGEFESGMEDLNREPAELEHVLRQQMSPVREADIGPLLIERQTRPVAWYEVELADLDPAEDGDVLEAIESVKTSNYGS
YEEHQSFRLDKEDGRIRIWLFADSPSQQDWAKDELPDGFDVLTEDEWEYVYGGGCRTLFPWGDSFDYHMKVRHFGELDRE
QKEQGEPVVSDRPFDLEKSGGMGLQFLGDPYKQELTISPEGLVRGKGGDGGTLICGGMGVTWGYFPVATYYRDPYEEELE
WEDRAEYLHYRRVVHL
>lcl|NZ_CP026520.1_prot_WP_042226074.1_3264 [locus_tag=PC41400_RS16530] [protein=WGR domain-containing protein] [protein_id=WP_042226074.1] [location=3622996..3623703] [gbkey=CDS]
MKKAFTYQDEKSNKFWTIDYSGLTIGVHYGKNGTIGKYEIKEFSSREDCEKEAKKLIASKMKKGYQENHIYDFDNCLYVD
VGEYGPHPKTSHPRFVAHFIEEFYYDCGDEEAPFGSDEGSDTLNEMQEIIRKKHTIDFATFPKHLIESLWGMHYIAVDTL
DEKTVRDIAKEHEMDMLQSDMVTYATAFGQIKITGKLDSDLKKNALLALKRFAIMYSSGKLTATQQTMYNNLLTF
>lcl|NZ_CP026520.1_prot_WP_042226076.1_3265 [locus_tag=PC41400_RS16535] [protein=SMI1/KNR4 family protein] [protein_id=WP_042226076.1] [location=3623843..3625207] [gbkey=CDS]
MSKELLDKLQSWYDDDQHDKIVEAIEQLPESERDYELVGHYGRALNNLGRYTEALQQLLSVEQQGKQDATWNWRIGYSYF
YMHQWEKALSAFEQFDRLEPGDADVDRYIEACHDIMVRQAVAIREITRVPFRERDFSQFWEMSDYADKSYVEAPPTAEMI
ASIEEELRYKLPADYIWLMQQQNGGVPVNTCFPTAVSTSWAEDHVAITGIMGIGREKAYSLGGSLGSNFMQEEWGYPDIG
VVLCDCPSAGHDVIMLDYRACGRDGEPAVVHVDQEAGYYVTFLAPNFATFIAGLVNEEAFDTSEQDKLDDLDKVEYAQFS
SLLQSLCEKADGIDRIEAVIRNICTQIVEDKGYFAFHADELSMLMYDIQFWLYAGANPGVTRAQYLAEYEKMIAFAQGFS
TGGYAPDFVSSWLDERIKQGDIVNKEGVLSFTAAATERLVGKLIAIESGLGDEG
>lcl|NZ_CP026520.1_prot_WP_164977134.1_3266 [locus_tag=PC41400_RS16540] [protein=ankyrin repeat domain-containing protein] [protein_id=WP_164977134.1] [location=3625218..3626039] [gbkey=CDS]
MTETKNNETLHHACYNQNVDMVREYVINAKANSLNKKLTDYSNPINGTPLQIACRVGNLDIVKLLIEAGADKEIKDVGRQ
SPLSIAISYNYFEITRYLIEAGADVHSKGPNNLQPIHFACERGSKKMIELLLSQGIDINRLDSLKSSLLDFTTNLSGGNL
EATKTLIENGIDQKYYSTAFKWACWRNNLKIAAYLLENGADYKSQTTPKSEMLFWICGLGYKEIVELLLGVGVDFTTQVK
FKGKMRSYSGSPLERAIETKQYEIVEIINSFKR
>lcl|NZ_CP026520.1_prot_WP_042226079.1_3267 [locus_tag=PC41400_RS16545] [protein=TniQ family protein] [protein_id=WP_042226079.1] [location=complement(3626460..3627977)] [gbkey=CDS]
MQPKRDLNVVWRKEWIHKYEAPWSIFEKILFANQTTRNDIIKAFGVDQVQKLRNMKKVGDVWKELYELKGIDEEILSNTL
DFDLHEQNKVTISLLLQPLQHFKEKPSSWFTNHLKWCTDCLQNGFHSWFHQFSLIEICPFHETKLHTRCTSCQEEIPFLL
SDRRLGSPFTCNCGYKLADFSNSRWREWDIAECEIKDSSLLRWLSINREEKHPCTKLFFIPQYGRIDLLVNTTPFASANF
QRKNKNSRHVTHVLNQEQLKTIFKANKETFKSIDRYIRKKLIKNHIHCINQLRDLRNQDYSKFPDICPHAYAYIFWRKSV
LQKTHFYREYMNADDLENPVMNFADIHVTTKIISEEFKYLSSQFLHHNSGTNVTQLIWLQNKFTTHFCLNYFRLWLQIAQ
IGAQSESVPKWDALNKLKQASLSKFSFKYIIKNDRLSILEAYRIQVNEAIPNMNCPNRALKQKKKINSMQSFIPLKVAMD
VFDKPTSENKQFMKYVDRFVSRLNF
>lcl|NZ_CP026520.1_prot_WP_053228605.1_3268 [locus_tag=PC41400_RS16550] [protein=TniB family NTP-binding protein] [protein_id=WP_053228605.1] [location=complement(3627979..3628962)] [gbkey=CDS]
MERPNIPAGTHPIEQGHYLLPTKEVLRLMKSISKIVNNRLPGMIVYGKPRLGKTSAVKFAMENLSLMVGAPLPILIANTN
WYKVPSEEKFYYDLLQDFRFPFPSKRKPAEMRRQIVNLMLEKAEKSKMRRIVLVMDEAHRLTEFHFNWLMDIYNELDREK
ISMTVISIGQEELLSRRTFFLEQKKSQIIGRFMTHEHHFMGLRSLDDIRFVLKCYDVPETSSYPKDSDWSYTKYFFPIGY
EEGNRLEQEAAHLYQLFVEIRKEHGVSTDFEIPMEYFAFTVETALKMYGAHGDSCRWLTVSQWRECIKLTGYIESEVYMA
LVHKGAN
>lcl|NZ_CP026520.1_prot_WP_042226083.1_3269 [locus_tag=PC41400_RS16555] [protein=hypothetical protein] [protein_id=WP_042226083.1] [location=complement(3628981..3630963)] [gbkey=CDS]
MSRSRLNENEGLLPEQVNPDLWPTIDEASLTEHSRKIYQNRKTAVLMYFKREKSLHEIKQVTTIDQQNLRRLIKRCMSSD
EMGGIWGFRALIPQKKVKAYQFNVLSKVSNKSRKTGEFNLLLETYPDLRDLIEDLYLGRNRRSLEPAMTPKSIHKRFIEE
CRRKEIPQSSYPFNTMWMGFRALRRYLNRLGQLHFARTAAFRYGQDAETKALHTGEGEQNYPSTLIPYQRVQFDAHRIDG
IYCVEINTPEGDTVPIILERFWILSLIDVATRCILGYSLSLAREISASDVMHCVRNAVLPHKKLKLTIDGLTYHSVGGFP
SAVFPETNSWAVWDVICYDNAKAHLANLVKDRLKSLIGCVTNLGPVALPMRRSLIERFFKTLEESGFHRLPNTTGSHPDD
PRRNDPEKKAIQWKITYDHLKELVDVLISNYNGTPHGGIHHQRPLELLAKRMGNGLLPRRLEETKRSEMLFLQTVIKRKV
RGSLSTGRRPYIQYEGVEYRSEKLNHAAYLINQELTLHVNVDDIRIVKAFLEDGSEFDYLTAAGKWCLSPHSLTTRKAIN
SLVTQGLLHYTSWDDPIFVYTDFLLKNANKGKRGAVNKFTHLKEVKREAVEQPAGRKQALDSVKQQHEALDKARELKQQL
QQEQEIESYEKMLSMFKTKS
>lcl|NZ_CP026520.1_prot_WP_053228606.1_3270 [locus_tag=PC41400_RS16560] [protein=TnsA endonuclease N-terminal domain-containing protein] [protein_id=WP_053228606.1] [location=complement(3630956..3631636)] [gbkey=CDS]
MFDLQKPKYKPIVLPRNKKYGNNCWLASGPKVGQRDVVLYSDLEFDHWVTVETNPNVLTYCEQPLEISYTLNGKLHSTIF
DMWILMKDGTERFVEIKYEKDLKPDGRSYIRTKRQITTQKEWCEQNGLQHEVITEKDIRAGRFSIDNKLKMLLNYINHDK
PASVDLVKNYMTKQKKAVGDICTQLKGKCSTYEVLLACQWMCYSGIASADLKEQIWGNKMEVWSLE
>lcl|NZ_CP026520.1_prot_WP_084706486.1_3271 [locus_tag=PC41400_RS16565] [protein=helix-turn-helix transcriptional regulator] [protein_id=WP_084706486.1] [location=3631772..3632005] [gbkey=CDS]
MEEEEFYQWLGQYVKNIRKQRNKTQVEISSIFRLSRSSLANIETGRHRISCYTLYELLCALDHPFNDVLDSIPTLPK
>lcl|NZ_CP026520.1_prot_WP_042226085.1_3272 [locus_tag=PC41400_RS16570] [protein=hypothetical protein] [protein_id=WP_042226085.1] [location=complement(3631982..3632299)] [gbkey=CDS]
MKEIHMCCGIANVTNAGLQFNGLYYSNPTLIKLDWFLLAEMQGSWPIPILFDLLDNRHILLLDFDHFEVATTIEKDEQHL
DPEIVDAYQQAFRNLKTLLFWQRRY
>lcl|NZ_CP026520.1_prot_WP_042226088.1_3273 [locus_tag=PC41400_RS16575] [protein=helix-turn-helix transcriptional regulator] [protein_id=WP_042226088.1] [location=3632430..3632777] [gbkey=CDS]
MMDNSVLKRIGQRVRDLRKERNLSQEQLGELAGVHFSYIGGLERGQKNISLLNLDKIAQSLDVPLSELFLYGRHLKRTIK
SGKEELINEIHESLMSLNESDLKKLQRLIEELFTK
>lcl|NZ_CP026520.1_prot_WP_042226090.1_3274 [locus_tag=PC41400_RS16580] [protein=tyrosine-type recombinase/integrase] [protein_id=WP_042226090.1] [location=3632926..3633972] [gbkey=CDS]
MKIQELKDLHLLETRMETFPWFVHKFIHYKLDTNLSPSSLLEYSRDFESFFSWIIQIKSPDNLNYITQMDIGQLEVLTAE
NIQEFEEYLLINSMKERTILRKIQSLRSLFNYLHDVAENDAGEPLLSKNIFRKITKRRTGSPIETARDIQNKVLASNESD
DFIDYLRSRYRTENGSNLQAIWNYDINGVRDICIINFMLKTGLLVSDLINLNVSDISIHDQQININRQWSGIRSNHSVFF
SLSTKTDLLNYLKIRDTTYRPNHNEAALFLSLPNGQKIGKRITKRGIQSMVIKYATKFGKPGLTTRQLRHSFGLEHQRQS
NYVRTKQQLAYRSTEATEKYQFLTDFVD
>lcl|NZ_CP026520.1_prot_WP_053228608.1_3275 [locus_tag=PC41400_RS16590] [protein=HigA family addiction module antitoxin] [protein_id=WP_053228608.1] [location=complement(3634626..3638696)] [gbkey=CDS]
MRNSLEASGHPGTYVRQMIIPSDMSVKEAAERLGVGRPALSNFLNGKSSLSPQMAVRLEKAFGADRKKLLEMQASYDRKI
RVTTDKEITVRAFVPVFLTIKARQIEDWVDSQIDARHHFPVLLRKLVHSTSNGLRQIDFPGYDNAQRKGNDGYVESSVAT
PWVPEGSSYWEFGTDKNPGGKAESDYTARLKSVDPSVRANSTFVFVTPRNWHAKKTWEERKNKSGDWKVVRALDASDLEQ
WLEQSVPAQVWLAEQLAMPVKGFETLEQAWSRWANASEPPLTSEFFSPSITASRDTFKTWLNKPNNRPLVVTADSSDEAL
AFLACLFDDSEFCQFKDQAAIFTSPEALRTLIASFVPFIPIVYSENVERELVDAHRRLHCIVFRPRNAADTEADIKLDLL
NHEAFKNALTSMKIEEGDIDRLERESGRSPTILRRRLSQNTAIRTPVWACDDNRTEAIVPMALIGAWHAGTEADRKIVSN
IANREYDTIELDISRLLRLDDSPVWSAGHYRGVASKIDALFAIARMVTPADINRFFVAAEYVLSEFNPALELPEKNRWAA
ALYDKTRDHSSALRTGICETLVILSVHGNNLFQSRLGIDVKGRVAVLIRRLLTPLTLEKLLSHDRDLPRYAEAAPDEFIK
IVEEDLRTDNPIVFGLLKPVDSSLFGVSPSRTGLLWGLECLAWKPQNLPFVSAILARLSRLKIDDNWANKPDASLQAIFR
SWMPQTAASVEQRIKVLEMITKRFPDVGWEICIEQIKPGSRIGHYSYKPSWRGDASGAGQIVNRKEMYDFNRKALDLLIA
WPSHDEKTLGDLVESLEGIPEEDQNKIWDLIDEWSQSAGESAKAELRERIRLFAFTRRSRHRKLGKATRDRAREAYERLQ
SENPVIRHGWLFAEQWVQESIEEIEEEDFDYRKREERIDRLRQEAIKEIWTKLGFQGIKELIACSGAAGVIGRYASLCVI
GLKTRVDFIQRCLSLEGDSRSKAEWCVQGFLSAIGDDSCAEVLQKAAEELIEEERIRLLICAPFQASTWRLLDGYGEGVH
AGYWKNVFPSWGRHMPAELTEMIDCLLEAQRPRAAFHAVHMNFKDIETSRLKRLLRDVVTVDAEPADHFKLDSHYISKAL
SSLDGRSDVTREEMAQLEFLFIGALDDSEHGIPNLERQIAESPVLFVQAVALAYKRRDEGEDPQEWRVQNPEQRAAVALA
AHRLLDRITMIPGTDENGKIDAVALASWLAEVRRMCRECGRADIGDQCLGQLLSRTTVTEDGLWPCEEVCEAMEGLASPQ
IGKGFYIGVRNSRGAHFRGEGGEQERELASKYRTLAERVYFDYPYVGKILEDITKSYEHDAKWQDSEANISKRLRD
>lcl|NZ_CP026520.1_prot_WP_042226093.1_3276 [locus_tag=PC41400_RS16600] [protein=hypothetical protein] [protein_id=WP_042226093.1] [location=complement(3639844..3640896)] [gbkey=CDS]
MKSSLYYHPQRVIVKDSAIILNSAHYMPIKHFVGLLEEKVPQCIHDLKNLTVQYSIATEWSLEKGLTGLEDPSLWEIVNQ
SDENNSFAGYLPLKKGLIEWINKYKLFSSTGFYLSLALWSVEVYYKQCLETEREKRIAHMKKTCEELEISEAQYWVSFPY
EPNKEVLSLSKEIVYANERDTEALYESTENITEFDDLFFDQNFPFIFAPNQLYLILNGDPSIDDFEDLKPHSKNEEDYKP
NPANSFSGYGWDPREDSWREYEKRIDCWYEKYKHMYKERTESILYENGYVRSKAKRNIEHYSWLVYYQILGWSLKDIANH
WSEKNESAFSEDTIWKGLKSAAEIVYINLR
>lcl|NZ_CP026520.1_prot_WP_042226095.1_3277 [locus_tag=PC41400_RS16605] [protein=helix-turn-helix transcriptional regulator] [protein_id=WP_042226095.1] [location=complement(3641216..3641536)] [gbkey=CDS]
MSETVLIRIGEIIKELRKSKGMTQEELAELMATAPQYVSALERGEKNVTINTLVKAAEALNIDLFDMLCLGKGDPAKILD
LSAIILHKPETEKEKLISIFKSISSL
>lcl|NZ_CP026520.1_prot_WP_164977135.1_3278 [locus_tag=PC41400_RS16610] [protein=hypothetical protein] [protein_id=WP_164977135.1] [location=complement(3642102..3642773)] [gbkey=CDS]
MRNYFKTNDEQLLYGIVNDIVQEINAVFRFSPSDEACTTLAYGVNVYQLTLRGLAGYKESYYSVADTYAIRLAEFSAGFI
NPIKESMQAEPVDVIFISKKSESICTVKEDPFGRESSESGAEYSFSLSFKNTMAVPHILGFAHSYNAYTTCVPYPKGSFG
HYNFNWKPDRNFFKTTEEYKQTEAFRKRVYDARQEQAGSSSNLWIERSRSGKKPRFLFDPISL
>lcl|NZ_CP026520.1_prot_WP_164977136.1_3279 [locus_tag=PC41400_RS29850] [protein=hypothetical protein] [protein_id=WP_164977136.1] [location=complement(3642782..3643021)] [gbkey=CDS]
MPEGSYEDILYAGLEATTVAFGTSFGGVLGERSQGKLLGLCFLEGKKTSLLRLAGNCLKVSGRPLMRRFCGITPGKLLV
>lcl|NZ_CP026520.1_prot_PC41400_RS16615_3280 [locus_tag=PC41400_RS16615] [protein=WYL domain-containing protein] [pseudo=true] [partial=3'] [location=complement(<3643584..3644156)] [gbkey=CDS]
MKLDRMLGITMELLTRRRVTATELAARFEVSIRTIYRDVDLINQAGIPVASFTGTDGGFELMNGFFLTKQHFSVDDFSVI
YNLLKGMKGAMGGQSVLLMQKLSSLQPALLNGGYHDKIVFDMSTSEREKEIVHPLLHPINRPRIIAFSYTSATGTLSQRK
VEPIKLYWERGAWYLEGYCLSCQAKRFFRVS
>lcl|NZ_CP026520.1_prot_WP_042226101.1_3281 [locus_tag=PC41400_RS16625] [protein=hypothetical protein] [protein_id=WP_042226101.1] [location=complement(3644980..3645231)] [gbkey=CDS]
MIQSLRWVLITSGIFLVGLAGLEKIILFSAVFNKTHAMDKDAILINIPKYFWNITNYTGYFGIIMLVAGIAIVVYSKVKD
IKH
>lcl|NZ_CP026520.1_prot_WP_042226102.1_3282 [locus_tag=PC41400_RS16630] [protein=hypothetical protein] [protein_id=WP_042226102.1] [location=complement(3645311..3646096)] [gbkey=CDS]
MNVFSRALLRTGGVLALVCVLGLSLTNGSAFAQKKVDERYSIVVINGKEVIRDSVDKVYISSPVLADGTQLTLQNYKNIL
EESDFIQEKVEREKANAPISRQDESPTQDYSPSAIYQYYSYDEDLGWQATGSKVQVSAWVDCPSGTTACTIQSTNQKTIS
HSFSTNVESGWKETIRVGATYQFQYQLATSTSYTLPIEPGKTGYMSFRPWINYTRGYVTTSTYRDGWFVSESKSGMVFAN
GPDKLSSGEANGIYAIEYPYN
>lcl|NZ_CP026520.1_prot_WP_042226104.1_3283 [locus_tag=PC41400_RS16635] [protein=hypothetical protein] [protein_id=WP_042226104.1] [location=complement(3646947..3647435)] [gbkey=CDS]
MDEKLIKLAGSDMRNAMKEEIKQNQRIIDENLIKRNSPRYLKEAVFYAIARNTSINHFLRLDEPTPLRLKDSNEISAEFN
SLMYKVGAVKYPLEVKRKAEYDLFYDYTHVDYKKRRHFIDKELPIRPKTIEVLELGINALGFCFIIIFRFWNMLSDDKNI
YA
>lcl|NZ_CP026520.1_prot_WP_164977137.1_3284 [locus_tag=PC41400_RS16640] [protein=MFS transporter] [protein_id=WP_164977137.1] [location=complement(3647709..3648800)] [gbkey=CDS]
MLSLVEVGILLSVNRFIRLPLNPIISWLYSKMSIRTGLFISLAIAGITTLFGWAQGFVFWLILRCIWGIAWSILRLGAYY
TIVEVSTNDNRGQNMGSFNGLSRIGGLVGMLGGGFFIEWFGIRNVTLVFGLLALVALPLILRIPKAESQESKNGNSLNSL
TTLIKNPLLLKLLLTVFLVMLCLEGMISATLSHLIDVRGVTIDIKGVILSAAILASLIQASRLIIGVFISPWIGKKSDGE
WGRRRILIVGLLLATLFVVLTQLNIPDYLWLINLKATLLTTSILIVLLNSFASDLAEERLKAATITLYVIISDIGAAFGP
VLGYLSEQAFGLVTTYLISAAILLVLSLQWILLNVTKKSISRS
>lcl|NZ_CP026520.1_prot_WP_241688903.1_3285 [locus_tag=PC41400_RS16645] [protein=tyrosine-type recombinase/integrase] [protein_id=WP_241688903.1] [location=complement(3648930..3649136)] [gbkey=CDS]
MAKEAGVRANNNKVTPHKLRHTCATVLYNDGEGADLLGHTDPKTDTIYTKVNIEKMRKIVTSNPLELT
>lcl|NZ_CP026520.1_prot_WP_042226105.1_3286 [locus_tag=PC41400_RS16650] [protein=hypothetical protein] [protein_id=WP_042226105.1] [location=complement(3649412..3649672)] [gbkey=CDS]
MNLKSLAIGAVALGLLTGGGYLTTAYASSNDSVKSEVQLAKVDIKTFGPGEMPEIPAGATEMEKTKLDLETAKVDVKTFS
GQAQSG
>lcl|NZ_CP026520.1_prot_WP_042226108.1_3287 [locus_tag=PC41400_RS16655] [protein=HPr family phosphocarrier protein] [protein_id=WP_042226108.1] [location=complement(3649976..3650242)] [gbkey=CDS]
MNAVKIIMPTDGFHVLAAKQISGAASAFQSEIRIIWEEKNIISDAKSILGIMAMEIKKGTSLTLTFDGPDEEQAMELLIL
WFEHNANN
>lcl|NZ_CP026520.1_prot_WP_042226109.1_3288 [locus_tag=PC41400_RS16660] [protein=glycoside-pentoside-hexuronide (GPH):cation symporter] [protein_id=WP_042226109.1] [location=complement(3650266..3651603)] [gbkey=CDS]
MQTENIKFREKLGYGVGDLAGSFIWAASGMFLTFFYTDVVGISAAVVGTLLLVARIFDAFVDIGVGALVDKTNSKYGKAR
PWLLWFAIPYGIAGVLLFSAPDLGPTGSLIYAYATYLLVNVMYSAVNIPYGTLNSLITQNAYQRSLLNIFRMVSALTGTL
IVMTMTMPLVQTFGGGKGGWFGTFALFGTLAAFLFLITFRSTKERVKPSVVQKNVPFKKSIKGLFRNKYWLLMVIIMLVF
NISSSVGTAVTIYYAQYILQNKDLVGLLGAVSIFPLFIGLLFIAPIIKRFGKRNASIGGLIIVLLGSIFMLINPENLTVI
IISLIVKAIGSVPMVSSLFAMLADTIEYGEWKTGMRNEGLVYSAGSFGSKTGSGIGAALVGWLLALGGYLGGQDTISGNA
SNMIEFLYIYMPIILNAILLLLLIPYKLDEEYPQIIEDLSAARNI
>lcl|NZ_CP026520.1_prot_WP_042226110.1_3289 [locus_tag=PC41400_RS16665] [protein=glycoside hydrolase family 3 C-terminal domain-containing protein] [protein_id=WP_042226110.1] [location=complement(3651626..3653920)] [gbkey=CDS]
MKRDIHALIAEMTLEEKAGMCSGAGFWHLKGIERLGIPSVMVTDGPHGLRKQQGESDHLGVFKSSPATCFPSAAGMASSW
NRELIGQVGVALGEECQAEDVAVLLGPGGNIKRSPLCGRNFEYFSEDPYLSSEMATYHIQGVQSQGVGTSLKHFAVNNQE
YRRFGIDAIVDERTLREIYLASFEGAIKNAQPWTVMSAYNKVNGAFASENEYLLNQVLKEEWGHEGFVVTDWGANNERVD
GLAAGQELEMPASDGVNDRKIVKAIQNGQMSEEVLDRAVERILRIIFKYVDQKKPNAEFDLDEHHRLARHVAGETMVLLK
NEDHLLPLAKGQKLAVIGSFAQKPRYQGGGSSHVNPTRLDIPLEEIKKLADDPASVIYAQGYHLDRDDSDAALVEEAKQA
ALAADVVVIFAGLPDRYESEGYDRKHLRMPANQEQLIAEISKVQSQTVVILMNGSPIEMPWLAHAKSILEAYLGGQATGG
AIADLLFGEVNPSGKLAESFPIKLSDNPSFLFFPGEGNRVEYREGIFVGYRYYDKKEVVPLFPFGHGLSYTTFEYSNLYI
DRKEIKDTNTVNVQVTVKNIGTRAGKETVQLYVSDIASTVVRPVKELKGFAKVDLQPKESKTVSFTLDKRSFAYYNVEFK
DWHVESGDFDIMIGRSSADIVLSERIYVESSVVLKIKVDRNTLIADLLGDPQRAPIIQGLLHNLQEKMDGMFGGDNVTQS
LGEGAKDMVAEIMNVVPLRMLSFTSKGEISDQLLDNLIRQLNEC
>lcl|NZ_CP026520.1_prot_WP_042226112.1_3290 [locus_tag=PC41400_RS16670] [protein=helix-turn-helix domain-containing protein] [protein_id=WP_042226112.1] [location=3654036..3655271] [gbkey=CDS]
MNKAHFSFDLTQSCQIIRESIQLPVTCYGIGEPLELTGEQDTFYRGFLMGIEKLTHSFSAKMAELPILWTTNAWEQFIVL
PVKRNEGLQQFILIGPAIEQTPPDELIHGFMSDNRKPYHELPHWKAYWSYLPIVSRLRLIHISVLAHWIINQEVVEISDV
AQYSCQYVWPATFESDVEKIISEQRESSITQHSSEIEKQLSELIRTGNTTELLKMWSSSPQAGVGVLSKQSQLRNVKNLA
ICCVAFAMRAAVEGGLDEELAFTLSDWHIQQIEELNKVQAVQAVLSRAVMDFTERVAKSKHESMSRPIRICCEYMVNHLY
KEITIEKLSELTGVNSNYLMNLFKEQTGMTLMNYLQEQRINEAKKLLKITNDTISSIGTRLKFYDQSHFTKVFKKLTDMT
PRQYRSTFQTI
>lcl|NZ_CP026520.1_prot_WP_042226115.1_3291 [locus_tag=PC41400_RS16675] [protein=2,3-butanediol dehydrogenase] [protein_id=WP_042226115.1] [location=complement(3655927..3656973)] [gbkey=CDS]
MKAAVIYGTKDVKIEEVNEPSVSKNEVKVKVSWAGICGSDLHAYHYGLGIPIEPNELTGVQAPLTLGHEFAGTVVEIGEG
VTDIVVGDRVAIEPLLYAEDDVYVRQGQYNLANRFAFMGIQSNGGFAEYAVMNESKVHKLPENVSLEEGALIEPTAVAFQ
AIRESQLKVGQKVVVFGVGPIGLLTIVCAKAAGAAEIIAVDISNDRLALAKEVGATYIINSMEENATEKILSLHKHGVDI
AYEAAGAEITFTTALNVLKKRGQLMVIAAFGKPISLDVTQHLLMKEINITASFAYCNIFPEVINLIASGRMDVKKVITKK
IALDHLVEDGLELLEKDKSHAKILIDIK
>lcl|NZ_CP026520.1_prot_WP_042226116.1_3292 [locus_tag=PC41400_RS16680] [protein=SDR family oxidoreductase] [protein_id=WP_042226116.1] [location=complement(3657012..3657698)] [gbkey=CDS]
MGLLKNRVVLVTGAGTGLGRETAIQLAAEGAEIVLTGRRMEKLQEVAEVIEQAGGRSLVFSVDVTNAEEVNRMRDQLIAQ
LGRVDVLINNVGGTGDYSTVHDMTISLWDHTIRLNLYSSFIVTNAFLPVMRKQQYGRIVSITSAMANYVYEGLGAYSVAK
AGLEGLMRTVALEEEKHGILVNMFDPGNLKTEQNPHGEGDPATVVDGIVKLATLPEDGINGQVFRANQ
>lcl|NZ_CP026520.1_prot_WP_241688670.1_3293 [locus_tag=PC41400_RS16685] [protein=MarR family transcriptional regulator] [protein_id=WP_241688670.1] [location=complement(3657712..3658080)] [gbkey=CDS]
MEKLAEEAFASIDIPPSYTYLIITLNDYPGITQKELCEKLTIAPSTSTRFIDKLVQRNFVERKPDGKHILIYLTDEGKSL
CKEIYENLNKIYLRHSEILDKKHSQQLIHLLHEASQLMEKNK
>lcl|NZ_CP026520.1_prot_WP_042226120.1_3294 [locus_tag=PC41400_RS16690] [protein=MFS transporter] [protein_id=WP_042226120.1] [location=complement(3659506..3660714)] [gbkey=CDS]
MLMQWKQKFAFIFSGQIFSILTSSMVQFSIIWHLTETTGSASVLMIAGLAGFLPQALLGPSIGVWLDRWNRKLTMIVSDS
VIAVSSLVLGAYYLWGEPSLWVVYAILFIRSIASSFHAPSFQAAIPLVAPEEQLTRVAGWNQLVFSASSVLGPALGIAVY
SATSLGTVLLLDVIGALIANLMLLLVTIHQPKPEKLPTSSFRREFTLGWKAFLSEKSIVTITIAMAIFSVVFMPLAMLFP
LMTLSYFGRGGYSASLIEAVFGIGMILGGALLSVFASKWKDSTYMSLSLVAIGLACVLSGVVGSDAFLAFIFLSLLMGAA
APLFNGPYMAMIQKSYEPEILGRVLSFVTSIGLLSSPIGLALAGPVADRFGVQVWFFWSGIVMALIGIFIFVKFYGISKE
EK
>lcl|NZ_CP026520.1_prot_PC41400_RS16695_3295 [locus_tag=PC41400_RS16695] [protein=Msr family ABC-F type ribosomal protection protein] [pseudo=true] [location=complement(3660766..3662226)] [gbkey=CDS]
MLLIKATDIHVEFFGRDVLDIEQLELYEFDRIGLVGANGAGKSTLLKALLGKIPLQHGKIAREGTFTYIPQLEEAVTQDV
QDHALAGKLGVSQVHVDTMSGGEETRLKIAQALSGEVHGIFANEPISHLDRESIDFLVNRLKHYSGALLIISHDRYFLDQ
VVDKIWELKEGKITEYWGGYTDYLAQKEEERNSQAAEYQQYIVERERLEQNIEEKKSQARKLDQQSKGADRKNRTNSGGR
LGHQKSQGSKQQKLHNAVKNMERRIEVLGVVTAPETIRNVQFRLSKALVLHNPFPVVGNEISKWFGDKVIFDNVSFHFPL
GGKIAITGANGAGKTTLFNMIRDQESGIKLSLKAEIGYFVQTGYKFDRNQNVMAFMQEDCEYQV*DIRAVLASLGFTHQD
VRKELSVLSGGEIIKLQLAKLLLGRYNILLMDEPSNFLDIPALEALETMMRNYLGTIVFISHDARLLESVADQIYEVSDG
RLVRLK
>lcl|NZ_CP026520.1_prot_WP_129112498.1_3296 [locus_tag=PC41400_RS16700] [protein=hypothetical protein] [protein_id=WP_129112498.1] [location=complement(3663682..3664233)] [gbkey=CDS]
MYIVYIPSASENRLNDPISNSLSVPVVEVTIPATSELINPNVPIHKLSIIPPTGLNPGGFTPFGEDAPTGNTATNLADND
YNYSVDKLGYKVYTNAFFTGVSKMRVSINNFKQLSGVATGKNITITLYKSNGSAVSSTTLDMSATTADYTNFSGLDTNTR
YYVLFSTKTNDNTYSFNGTISKG
>lcl|NZ_CP026520.1_prot_WP_156972772.1_3297 [locus_tag=PC41400_RS29855] [protein=hypothetical protein] [protein_id=WP_156972772.1] [location=complement(3664912..3665049)] [gbkey=CDS]
MVALAERLYFDYPYVGGILDEIAQSYEHEARWQDSEANISKRLRD
>lcl|NZ_CP026520.1_prot_WP_042226287.1_3298 [locus_tag=PC41400_RS16705] [protein=PhzF family phenazine biosynthesis protein] [protein_id=WP_042226287.1] [location=complement(3665225..3666103)] [gbkey=CDS]
MNVKVYTLNAFVKGERGGNPAGVVLEGGLSLCAAEMQLIAKELGFSETAFMEKSLFADYKIRYFTPASEVDLCGHATIAA
FGLMHSLGLAKEGTSYKIETKAGILDVDISSEGLVYLSQALPQFLERISCEEIAPSLGMDAEDLETGLPIQIVSTGLRDI
LIPIRSRKLLNEVQPNFDTITAISEKYDVVGYHLFTMDTPDGAAAECRNFAPLYDIPEESATGTSNGALLSYLYQHGQLS
LREVEHVMFRQGYSMNCPSEIKVGLRMSESGEINQVRVGGAVAGIERRHITI
>lcl|NZ_CP026520.1_prot_PC41400_RS16710_3299 [locus_tag=PC41400_RS16710] [protein=phosphotransferase] [pseudo=true] [partial=5'] [location=complement(3666708..>3667433)] [gbkey=CDS]
CYHAEINSEENEFQLWLEYIDGVTGLDLTGDMYEKAALELGRYQRKLYVEQPAVLQSLANLSHADLM*NTYLHYRSWPVV
YDYIRSEDCEFPQHVR*MLIDIDEHADEILTLIEKLPIVLCHRDFWVTNLIYADGKIALIDWDTSGWGRLGEDLASLIAD
EADIDHMVEYNLRCVPAYYKGFSEYTDVSPIADNCVYEIILLIFGYRLVEWYLHTEDDDQKTKHVHTLQKIYEMKTSTAQ
G
>lcl|NZ_CP026520.1_prot_WP_042226126.1_3300 [locus_tag=PC41400_RS16715] [protein=tyrosine-protein phosphatase] [protein_id=WP_042226126.1] [location=complement(3668288..3669124)] [gbkey=CDS]
MSTTRDGAYPGQRIIVLEGTHNFRDMGGYQTADGRKVKYGIFFRSDELTGLTEQDLAAVQALNIKTIFDYRDDYEAQKKP
DPVFAGVKNIRIAAIQADQASRINISGNAGNADRNQHVIVDMVKNGYFKQFRADTMMMGLYTKLPLGNPSYKRLMEMIQH
SDNLGLLHHCTAGKDRTGVGAALILLALGVPEETVMKDYLLTNETMKDFNGKLLSQLAEHVNEVELRNIEHLLGVKEQFM
EAAFGSIKKTYGNVDTYFSEEFGLTNQRRKALQSMYLE
>lcl|NZ_CP026520.1_prot_WP_042226128.1_3301 [locus_tag=PC41400_RS16720] [protein=glycoside hydrolase family 3 C-terminal domain-containing protein] [protein_id=WP_042226128.1] [location=complement(3669987..3672269)] [gbkey=CDS]
MKDRIKQLISQLTLEEKAGLCSGLDFWHTKGVERLGIPSIMLTDGPHGLRKQKGDSDHLGIMDSVPSTCFPSGAGLACSW
DRDLIRKVGEALGVECQAEDVGILLGPAANIKRSPLCGRNFEYFSEDPYLSTEMAANHIAGVQSQGVGTSLKHYAVNNQE
HRRMSVDAVVDERTLREIYLSSFEGAVKQAQPWTVMCAYNKVNGEYCSENKYLLNDILKEEWGLEGFVVSDWGAVDERVP
GLLAGLELEMPSSNGAGDRKIVEAVRNGSMSEDIVDQAVERILTIVFRAAENKKTVTFNQEEHHQLAREAARESMVLLKN
ERRILPLSKDATICVIGAFAQQPRYQGGGSSRIKPMKLDNIYEELAKKVSGQALMSFAKGYNLEDDEIDEQLVKEAVEAA
GKSQVTVIFAGLPDRYESEAYDRGHLRIPDNQMQLIDAIAEVQQHIVVVLSNGAPIEMPWIDKVQGVLEGYLGGQALGGA
IADLLFGDANPCGKLAETFPQKLSHNPSYLNFPGEGDTVEYSEGLFVGYRYYDKKEIEPLFPFGHGLSYTTFEYTDLKLN
RQAMLDTDQLQVSLKVKNAGDVAGKEIVQLYVRDVESTVSKAFKELKGFSKIALQPGEEKTVRFILDKRAFAYYNTDIRD
WHVESGDYEIMIGQSAAEIRLKETVQIQSTVLIRRTYTRNTTIGDLMADPYTADIVKQFLRGFKGNPLFSAANDEGTSEM
FAAILKFMPLRAAAAFSQGAFTEEMLNHMVERLNGTGSDM
>lcl|NZ_CP026520.1_prot_WP_042226132.1_3302 [locus_tag=PC41400_RS16730] [protein=tyrosine-protein phosphatase] [protein_id=WP_042226132.1] [location=complement(3672582..3673418)] [gbkey=CDS]
MSTSQFAAYPANRVIPFEGVHNFRDMGGYRTSDGRKVKYGLFFRSDHLTGLTEQDLTFLQTLNIKTVFDYRGDHESRIMP
DPVLPNAKNIRIPANTEDQHEQMNLPAAPEGEEQQGHLLKELVKNDFFKSFCAETYMTELYTKLPINNHSFKRLMETIQD
PDQLGLLHHCTSGKDRTGVGAALILLALGVSEQTVMEDYLLSNETMKSFNRNLLKRLAEHADEAVLQNLNHILTVNEAFL
SAALGSIKNTYGGYDAYFAEEFGLTKSRREALQSMCLE
>lcl|NZ_CP026520.1_prot_WP_042226134.1_3303 [gene=galA] [locus_tag=PC41400_RS16735] [protein=beta-galactosidase GalA] [protein_id=WP_042226134.1] [location=complement(3673538..3676018)] [gbkey=CDS]
MRTRENFDRNWMFHVGEVGKIVKTVKKAGMIAGLTNALGEERHEPFAIGEAARITQNVARELLGDPNFEIAQIYGSDTPK
DKITGWVGVNLPHDWRLEQPYNPERGGAFQGYLPNGVGYYRKVFQIPQEDEGKKIIIEFDGVMRNSSVWVNGCFVGDHLS
GYTSFHYDITDLLKYGDEEGDNVILVRTDTTGGDEGWWYEGGGIYRHVWLTKHDYLHVDRWGTFVRTERMDNERAAITVE
TTVSNEYPQPMKYGIRTSILNPDGELVAVTESDCETPGIDKQTVLQSVQVTNPLLWSLDMPHLYEAITELIHDGQIVDTY
RTVFGIRTVEYRQNGLYLNGKYVQVKGTCNHQDFAGVGIALPDSVHEYKIKRLLEMGCNAYRCAHHAPAPELLDACDRLG
MLVMNENRISESTEIKLDDLKSMLYRDRNHPSIFMWSLGNEEKISGSYQANRMLKRLQDLTKRIDSTRPVTAANLSQQGM
DIEVNGVNYAESIKGSQTIGEWFQSHPEAKVLNTENVSFFSARGVYEDDPEGGRCSSYGSRYTMLGKEVDVGNLGGAGGT
STPERSWRYFLNNRFSGGLFVWTGFDYRGETTPFQWPSVLSHFGILDYCGFAKDSYYFYQSIWKDEPIVHLMPHWNWPDR
LGQQVEVRVYTNCSEVELVLNGQSLGKKQAETEAGEITLSWQVAYEPGLLEAIAYRDGVMVAAERKETTMEAYSVHLEAN
RSELLADGQDVSMVTVSVRDQSGRVVPVAGNLIRFHVAGEGKLLGLGNGDPGCHESDKGDTRSVFNGYALALIQSLDEAG
EIRIEASSEGLVPAVLTLQSRKDLTN
>lcl|NZ_CP026520.1_prot_WP_042226137.1_3304 [locus_tag=PC41400_RS16740] [protein=MgtC/SapB family protein] [protein_id=WP_042226137.1] [location=complement(3676082..3676861)] [gbkey=CDS]
MTGAAANVWQLGHWDLFVRMVLAAVLGGLVGIEREWNNHAAGFRTHILVCLGSTTIMLLSIYGFSDFVAESNVRIDPARL
AAQVISGIGFLGAGAILRNGSVIKGLTTAASVWVVAAIGLCVGAGFLFVAFLCTFFVIISLYVLNKWEKHLLRHRKKHEV
KCTIIDMPGVLGKVASKFGEQGIQIANVKMMPDEHAPPEAGGMPTMQLCFTVKNHQEENIVQALEEITALDMVLWTESSY
LPALKPAAESKELKIDTAE
>lcl|NZ_CP026520.1_prot_WP_042226139.1_3305 [locus_tag=PC41400_RS16745] [protein=alpha/beta hydrolase] [protein_id=WP_042226139.1] [location=complement(3676953..3677966)] [gbkey=CDS]
MNIPLWHDQPALQGFNEGCPSLTPYLLEGEGPFPAVIVCPGGGYTHRAVHEGEPVAKWLNAIGISAFVLHYRVTPAQYPS
QLHDAQRAIRTIRHRGTEWNIDPKRIGMLGFSAGGHLASMAGTSFDNGNPQANDPIERYSSRPDALVLCYPLITMGEFTN
VSCKSVLMGERQNDRALIELLSSEKQVTEETPPTFMWITADDPVVQAENCLMFAAALRKFRVSFEMHLFESGPHGLGLAG
GHREAQAWTKLCEAWLKSRNFLLVERVIDEYTTVGQLLADDCSSRVLERYLPDLLASPKIDYVKAFSLKSLFNFSDPMFT
DEKMAAILKDLKSGAKK
>lcl|NZ_CP026520.1_prot_WP_042226142.1_3306 [locus_tag=PC41400_RS16750] [protein=SGNH/GDSL hydrolase family protein] [protein_id=WP_042226142.1] [location=complement(3677969..3678616)] [gbkey=CDS]
MTGTRTILCFGDSNTWGADPQNRGVRFPADVRWTGVLRKELGDGYDIVEEGLPGRTTVWADPIDGIMSGKEYWPTCLLSH
RPLDLVVIMLGTNDLKERFSVSALDIAMSTLSLAGLVKTTFSRPGITIVPQVLILVPPPIKETGLFAGMFKGGEEKSKQL
GEAFAMLSPYTDASIMDVSTVIVSSEADGIHFEAEQHRKLGSAVAEKIKSMLVEE
>lcl|NZ_CP026520.1_prot_WP_042226145.1_3307 [locus_tag=PC41400_RS16755] [protein=carbohydrate ABC transporter permease] [protein_id=WP_042226145.1] [location=complement(3678643..3679518)] [gbkey=CDS]
MPSSGKFYQWTINIVMLVFTFFCIMPFWLLLSASLTDDMEIIRHGYAFFPKAVSFSAYEYLWTNSANILRAYGITILVTV
VGTAVGLTLTALLAYPLSRTEMPFRKILSFYVFFTMLFNGGLVPTYLLYTNYFNFKNTILALIIPSLLMNGFFVILMRTF
FATSIPKPIIESAYIDGAKEWKIFAKIVLPLSTPVLATVGLFYTIMYWNDWFNGLIYITDSRYFSLQNMLNRILLDAQFL
QSSTEVVATSHLPLTSMRMAIAAIGVVPLLVAYPFFQKYFVKGLTVGAVKG
>lcl|NZ_CP026520.1_prot_WP_042226146.1_3308 [locus_tag=PC41400_RS16760] [protein=ABC transporter permease subunit] [protein_id=WP_042226146.1] [location=complement(3679533..3680474)] [gbkey=CDS]
MKASKMSVLRLEMRKAKKYRMLVLMTLPGLLYFLVNNYLPMFGIAIAFKDVNFAKGIWGSEWVGFKNFEYLFKTKDAYII
TRNTLLYNLSFIVINTTIAISLAILLNEIRKRILSRTYQTLILLPHLISMVIVGYLVLGFLDVEKGFMNKTLLPFFGLEP
ISWYSETKYWPWILTIVHAWKGVGHLCIIYLAAIIGIDQEYYEAAKIDGANRWQQIKMITIPLIAPVITIMTLLAIGRIF
YSDFGLFFQVPLNTGALMPVTNTIDTYVYRALVNMGDIGMSSAAGLYQALVGFVLVLTSNAVVRKFNKDNALF
>lcl|NZ_CP026520.1_prot_WP_042226149.1_3309 [locus_tag=PC41400_RS16765] [protein=ABC transporter substrate-binding protein] [protein_id=WP_042226149.1] [location=complement(3680663..3682201)] [gbkey=CDS]
MKKSMKSAAMLLISVLATSTLVTACSSGSSNQPNASGAPELSGSANPDLKPVELTMVFPVAGEQKDLKAVQEKINKITKE
KINATVKLVQITKGAWAQQTTLMLSGNEKMDLIVSGRGTYEQQVAKGNYLALDDYIAKYGQGAQKALDDLDPAYLKATRI
NGKSYGITTVRDFANDYGISMRKDLVDKYKIDTKAIRSLDDLDAVFKVIKENEPDVTPVIKFANSILGSPFVGGMLDPLG
DGLGVLPAHDNGMRVANWVETPEYAKLLDTTRRWYLAGYIAKDAATSNEVWQNLMKAGKAFSTFHHMSPFSQAANSLSVG
QDLIEVRLLPAVTTTTNITSYMWSIPRNAQNPERSMMLLNLMYTDKDFINLLDWGIEGQHYVKKSDDIIDFPQGVNASNS
GYYPYQNWLFGNMFLSYVFNGEDPDRNQKLAEFIKNSKKSKALGFTYNPEPVKTEIAALTNVRDQYALALETGTVDPAKV
LPEYIKQMKAAGIDKVIAEKQKQLDAWVAANK
>lcl|NZ_CP026520.1_prot_WP_042226151.1_3310 [locus_tag=PC41400_RS16770] [protein=response regulator] [protein_id=WP_042226151.1] [location=complement(3682359..3683933)] [gbkey=CDS]
MYRLLIVDDEEFTANGIKASVNWSKFGIIHVSVANNARQAKEKFSEHPFDMMICDIEMPQGSGLELYEWVREEHPWTECV
FLTCHADFQYAKKAIQLGSFEYLLKPAPPEELEQVMTEMIKKISKDRKSSSIVIERFWQDVLQQVIPSQPDKIMEAIGKH
NIPYTSRMKFLPILIGIQHWKKELSARDAQIMEYALTNALEELVIHQVATAHIVRIKANQLLAVFSTEHAIEASGEKFRE
ELIKRCGLFIETCNQYFYCHLSCYIGQPSEIHGVRDMVENLVALRKNQVNAVDRVVLIDEKMANETPFPMPPMRIWSELI
KQGDKKKLMAESQDFLESLKRIEGLNSKSLQQFYLSFLQMVLHTLQQKGLYTDDILADHFAPERISSAAESVKNLQDWVT
DVLEFAMTSIEALGSNETVVEKIKRYISANIDQEMSRQYIADYIGLSPDHIVKLFKKETGLSISEYILKERIGLAKELLA
KSETSISNVALAVGFANFSYFSTLFKKEVLMTPQEYRKRFSSGM
>lcl|NZ_CP026520.1_prot_WP_241688671.1_3311 [locus_tag=PC41400_RS16775] [protein=histidine kinase] [protein_id=WP_241688671.1] [location=complement(3683944..3685680)] [gbkey=CDS]
MRKPPSTIFNSLRFKLITGLMLIMLPIVLFLIYTNFYSIQVVRNQVAQSNSNMISLYMGLIDKSLADIDSYLLKYASEVT
GLHVLDRSPEIDIDLYKMERIWQFKQLVNNVTYYEGLDYFFIYSPINNDLVFAPKQTGSADTDNQSIKDAIVWLIDDGQS
AESFQTGKWSVFRANKKDYLLYIIKVGGLYIGAGVNTQDVTGPLNLLDLGADGRALLVDGNHKPLKDESFFREQGIDLSY
TPKSYRLSGFDEGYLVIGEHSGKGDFGLVAVVPDQSILQKLPYLQRIISLIAAGTVLILLLAFYFMRRVVLRPINRIVLA
MRRIKEGHLEMRIAKKPTSNEFELMNEMFNSMVTDIQKLKIDVYEEQLINQKAELKHLQLQVNPHFFLNSLNVVYYLAQE
RKYGLIQELSLSLIRYFRFMFRSHTDHVLVRDELGHTKNYLNIQKIRFPGSLTYAITVPDDLLDCCIPPLIIQSFAENTI
KHAVNTDEPTHIEISIERDNRNDEERLHIRIRDTGMGYENEVLEKLRQDIKLTTEEGEHIGIWNARQRLRLLYGGQAIIG
FSNDNGAIVDIFLPIMKG
>lcl|NZ_CP026520.1_prot_PC41400_RS16780_3312 [gene=arsD] [locus_tag=PC41400_RS16780] [protein=arsenite efflux transporter metallochaperone ArsD] [pseudo=true] [partial=3'] [location=complement(<3686060..3686392)] [gbkey=CDS]
MCCSTGVCGPSVDPELIRISSAVHHLKTKGIDISRFNLASEPDAFVSNQVINKLLAEKGTDVLPVVIVDDQIVQEKGDPT
NGELSLWTGIPEDELTKKPKDILRLPCLNDK
>lcl|NZ_CP026520.1_prot_WP_042226154.1_3313 [gene=arsC] [locus_tag=PC41400_RS16785] [protein=arsenate reductase (thioredoxin)] [protein_id=WP_042226154.1] [location=complement(3686479..3686898)] [gbkey=CDS]
MEKKTIYFLCTGNSCRSQMAEGWAKKYLGDSWNVYSAGIEAHGLNPNAVIAMNEVGVDISNQTSDVIDSTRLNTADLVVT
LCGDAVDRCPMTPPQVRREHWGFDDPAKAQGTDEEKWAVFQRVRDQIGARIESFAKTGE
>lcl|NZ_CP026520.1_prot_WP_129112499.1_3314 [locus_tag=PC41400_RS16790] [protein=arsenic transporter] [protein_id=WP_129112499.1] [location=complement(3686917..3688215)] [gbkey=CDS]
MSLVVLACIIFLITLTLVIWQPRNLSIGWSACGGAILALLVGVVTLQDVLDVTQIVWNATLAFVAIIIISLILDKIGFFE
WAALHMAKAARGNGLRMFIYVSLLGAIVAAFFANDGAALILTPIVLAMVRALNFDEKKVFPFIIASGFIADTTSLPLVVS
NLVNIVSADYFGISFLQYAGRMIIPNMFSLAASILVLYLFFRKSIPKQFNAEQLKMPSEAIKDPNMFRMSWIVLGVLLIG
YFISEFLSIPVSIVAGIIAIFFLLMARKSPIVPVKEVIKGAPWAIVFFSIGMYVVVYGLRNAGLTDLLANVLQATVDKGM
FAATMGMGFIAALISSLMNNMPTVMIDALAIDATNATGALKEALIYANVIGSDLGPKITPIGSLATLLWLHVLTSKGVKI
SWGTYFKTGIILTIPTLFITLLGLYLWLTIIL
>lcl|NZ_CP026520.1_prot_WP_042226157.1_3315 [locus_tag=PC41400_RS16795] [protein=metalloregulator ArsR/SmtB family transcription factor] [protein_id=WP_042226157.1] [location=complement(3688231..3688578)] [gbkey=CDS]
MEQTHVDDISIIAQTFKLLGDKTRLTIMKLLEHQECCVCELVEIINTSQPAVSQHLRKLKDAGLVKETRRGQWVFYSLNT
ENKQYEMVEKLIQQLPSQDHEFTLLENKGLRIICN
>lcl|NZ_CP026520.1_prot_WP_042226159.1_3316 [locus_tag=PC41400_RS16800] [protein=alpha/beta hydrolase] [protein_id=WP_042226159.1] [location=complement(3689032..3689805)] [gbkey=CDS]
MTTANINGYSMHYIDQGKGTAILFIHPPVLTSLNFTYQIQGLSPHFRTIAFDIRGHGKSQPSKQTVTYPLIVEDIKQLMD
RLSIDKFFLCGYSTGGSIVLEFLLTHPDRAWGGIVIGGMSEVNERWLRNKISLGVAFSKIGAVGTIAFSNAWAQAKDLSL
FRELFNDAKKGNAKNAEQYYQNSLKYNCTSQLQEIYLPVLLVYGEKDKHFHPYAKILHQRLPKSELVFIKKVDHRIPTKA
SEPLNELIKQFVHRFGH
>lcl|NZ_CP026520.1_prot_WP_241688672.1_3317 [locus_tag=PC41400_RS30340] [protein=class I SAM-dependent methyltransferase] [protein_id=WP_241688672.1] [location=complement(3689825..3690106)] [gbkey=CDS]
MLHHVSDEISSYMKEFQRILKPNGTIIVMEPCLCKKKPLCNRFMKWYNGEYIRNEEEYIRLFRDHNYDCKVIKRFRKCFL
YHELFFSVKPKSL
>lcl|NZ_CP026520.1_prot_WP_053228610.1_3318 [locus_tag=PC41400_RS16810] [protein=hypothetical protein] [protein_id=WP_053228610.1] [location=complement(3690446..3690982)] [gbkey=CDS]
MKNIEFKVVAKECCDPNECCVEPAETIKHEAEIANSELPIAIIGVGPIGLAAAAHLVNRKQKFILIEGGSQIAHNIREWG
HVRLFSPWQYNIDKAAQTLLLNQGWNAPPPDVLPLGIEVVEQYLGPLAELPEIKPHIMLDSQVIAVSRENIDKMKSANRE
QSPFVLYIENLYECFSTL
>lcl|NZ_CP026520.1_prot_PC41400_RS30345_3319 [locus_tag=PC41400_RS30345] [protein=hypothetical protein] [pseudo=true] [partial=5'] [location=complement(3690985..>3691174)] [gbkey=CDS]
PQSKLSYHLKLLLDVNLITMERIGKWNYYELNETEIKGLLSAIVLHLLPNISKKIDL*I*GDV
>lcl|NZ_CP026520.1_prot_WP_129112500.1_3320 [locus_tag=PC41400_RS16820] [protein=hypothetical protein] [protein_id=WP_129112500.1] [location=complement(3692016..3692273)] [gbkey=CDS]
MSKEYHLTTTRRVPLKTQRRIRNISPGLVGKVVVVRTLFNTFTGCLLSIRGNTIRLRIFSGIDRVFITIRIPIGIILDIF
RFPCP
>lcl|NZ_CP026520.1_prot_WP_084706492.1_3321 [locus_tag=PC41400_RS30350] [protein=sporulation protein YjcZ] [protein_id=WP_084706492.1] [location=complement(3692543..3692650)] [gbkey=CDS]
MGLGVGTGVKGGIFTSTATILVLFILLVIVSRTFI
>lcl|NZ_CP026520.1_prot_WP_042226165.1_3322 [locus_tag=PC41400_RS16830] [protein=hypothetical protein] [protein_id=WP_042226165.1] [location=complement(3693063..3693476)] [gbkey=CDS]
MTDYNGDLELKYPFTVKPWSGSSFPTPQPVELAVNQVGEANEGQLVKAKNVWITGEYGSGDGGGVTVTHGDARRLCAGPF
GKRRMKHYFTVQENVAVQAGRQYVCGGQRGQSVRRGYRRREYRQDDAIGRRRTFGNC
>lcl|NZ_CP026520.1_prot_WP_241688673.1_3323 [locus_tag=PC41400_RS16835] [protein=cation diffusion facilitator family transporter] [protein_id=WP_241688673.1] [location=complement(3693659..3694543)] [gbkey=CDS]
MNSSKIAFLSVFSNTFVTLLKLIVGIVTGSVAVISEAIHSALDLVASLIAFFSVRLSGRSADKNHPYGHGKVENISGTIE
TLLIFVAGIWIIYECVHKILNPTIIELPFLGIIVMVVGAGINFIVSRKVKKAADEFHSVAMKSNALHLLTDVFSSLGVAV
SLLLVTLTGWTILDPIIGIVLALYIMREAYKLMKESFPPLLDARLTEEEESEILRVIELFKERYIEVHDLRTRRSGPEEY
VDFHLVTPSTMDVLSAHQLCDEIEQAIKAVFEHAQVMIHVEPDHERLGNQPLSL
>lcl|NZ_CP026520.1_prot_WP_174446772.1_3324 [locus_tag=PC41400_RS16845] [protein=DUF2935 domain-containing protein] [protein_id=WP_174446772.1] [location=complement(3695242..3696066)] [gbkey=CDS]
MANEFVARSLDEVRFWSRIMKEHSLFLKLGFRCEDTQLINEANQFYSVFESIENRANAFTVETDPHTIVRFNEEVHNAAS
HIWVFKRKVLGLILRCQLPGANNFPLLVDHVSREANYFRNRLTELNTGRLEPLPDAIIDENVFFLRIMADHAKFISHLLD
PSERKLVDQANNFSHDFDQLLFQARDLESMRPQSQTTPLLDQFLDQNRVSVKALRDFKKTARDLIDACRVKSIIHPLLAD
HVFREAERFLHIIDMFETSLTGRPVSTLHSMFED
>lcl|NZ_CP026520.1_prot_WP_241688674.1_3325 [locus_tag=PC41400_RS30355] [protein=hypothetical protein] [protein_id=WP_241688674.1] [location=complement(3696887..3697237)] [gbkey=CDS]
MLWIVFSTKTKAKEFAPWLLFWYWIRNILCQLKHHLLYGVHVVRCSCLPKGIPPKGIPPKGIPPKGIPPKGILPKGIPPK
GIPPKGIPPKGIPPKGIPPKGIPPKGIPPKGIPPKG
>lcl|NZ_CP026520.1_prot_WP_042226170.1_3326 [locus_tag=PC41400_RS16855] [protein=ABC transporter ATP-binding protein] [protein_id=WP_042226170.1] [location=complement(3697921..3698691)] [gbkey=CDS]
MSLLIADQLTKTYSTAGLFRSKASNVSGISGISFVIEEGMCLGILGESGAGKSTIGKIILGLLRPDSGSVTFQGTNLYTS
NRAQQKKWRRDLQVVFQDSYSAINPRMTVRQIIAEPLRNYERMSLREEEHVVCELLETVGLSSNDMRKTSKEMSGGQLQR
VNIARAIALKPKLIVLDEPVSSLDMIVQKQILIHLNELKEKYGLSYLLISHDVLAVNFLSDHVAFVDGGKIVETIETAHL
FNCMHPVSKRMVLSAF
>lcl|NZ_CP026520.1_prot_WP_053228611.1_3327 [locus_tag=PC41400_RS16860] [protein=MFS transporter] [protein_id=WP_053228611.1] [location=complement(3698688..3699878)] [gbkey=CDS]
MKPAIKENPFNQSAIWLYIVAFVFYATNHMLIIALPFLSQSLGAGRSEIGIIMGAYMFVSMFLRPVAGAVVDRFGTRNIF
MIALVLNVIVLISYTAENLWAFAIMRALQGAILAFFSMTIHLLVMDLLSDKVRGQGLSLLSLASMLPYTFVPALVLFLKN
QVSMTELLLLFAGLGFCNIFLGGRLFRSLSNNTGKQEKKEMEPVSIEAKAKAGRALIFPSLVMLLASIIFCIAPTYMPLY
LESQGLSTAPLYFLTETGVLIFIRFFGRKYIPSSDSFPKGLLVALVICFTLSPALLALSLSTPVLLIAAVCNGMALSLLY
PTLMTYISFVVPEKVRGYSIGWFIAAADFGTSSGAFLMGWLADAFTYRGMFVAAVGIGVFTLVLTFWHKGQKAVSS
>lcl|NZ_CP026520.1_prot_WP_042226172.1_3328 [locus_tag=PC41400_RS16865] [protein=ABC transporter ATP-binding protein] [protein_id=WP_042226172.1] [location=complement(3699847..3700704)] [gbkey=CDS]
MSSRSIVLEVNSLHVSMQTNRGLVHHAVQDSSFQIEAGKVLGIVGESGCGKSTTCMAILGLLPAPAQITNGSIKLRGKEL
VGLAQQQMRHLRGKELALILQNPMTAFNPTLTIGKQFTETLRAHQALSRTQAEQKAMDAIRQMGLNDAEKIINQYPFELS
GGMLQRIMIALSISMKPSLLIADEPTTALDSVNRYMVMKTFQKIKEEGHTAILLVSHDLGFIRKLADEVIVMKEGRVVEM
SDTEQILSSPQNEYTKMLLDARLTADSRIEQESKPSETGHQRKSV
>lcl|NZ_CP026520.1_prot_WP_042226174.1_3329 [locus_tag=PC41400_RS16870] [protein=ABC transporter permease subunit] [protein_id=WP_042226174.1] [location=complement(3700701..3701558)] [gbkey=CDS]
MVNIRNLIWATSRQRILFICLFFLSVIVLAGLFAPLLAPHDPNFVDLANKLKGPSAEYPLGTDHMGRCVLSRLLYGTRVS
LSLALLIVSFCLLIGLVIGSLAGYAGGKLDSLIMRCCDAILAIPALILAISLIAIWGAGWKQMAIALIVVQSVYYARFIR
GLVFNLKKQTFIVAARVSGTSPWRMIVRHIIPNLLPPLLTVVTLEIGWVIMDISALSFIGLGVQSPTPEWGAMINEGKSF
IGTHPRLMIAPGFIIFCMVALLNLLGEAFNDKKGLVRSFRKGGAR
>lcl|NZ_CP026520.1_prot_WP_241688675.1_3330 [locus_tag=PC41400_RS16875] [protein=ABC transporter permease subunit] [protein_id=WP_241688675.1] [location=complement(3701561..3702511)] [gbkey=CDS]
MQMTLFLLKRFAAMVPVLLGATFVSFLLLHLSPVDPAKAYLSASGITPTEDTLNAVRTQLGLNRPFWSQFGHWLWQLGSL
DFGTSFFSKLPVTRELAARLPITIILAISSLAIAILVSIPLGIASALHKNRLIDHICRLLAFLGASLPQFWLAYLLIYFL
SVKMDWFPLQGADTWQHYVLPSCTLAFALIATYSRLLRTGLLEAQEQPFIMYAKARGLGSFVLVARHMLRYAIRPVVIAA
GMNLGKLFAGTVIVEQIFAMPGMGGFFLDSVYHRDYPVIQSYVLIMAVVFVIGNLAADVVQAYLDPRIAQMPRKGE
>lcl|NZ_CP026520.1_prot_WP_042226177.1_3331 [gene=nikA] [locus_tag=PC41400_RS16880] [protein=nickel ABC transporter substrate-binding protein] [protein_id=WP_042226177.1] [location=complement(3702529..3704169)] [gbkey=CDS]
MVYPYKKLHTAIFIVLIFSIILAGCTPTNNTKSEPSSSAKAPASTTTDKTLTISWPKDIGPLNPHEYTPNEFFAQGMYYE
PLIDYGEGGKLVPKLAESWDISEDGKTYTFHLHKNVKFSDGSDFDANVVQKNFDTILANRDMHSWLGLISEIEKTEVVDA
NTFKVILKNRYYPALQEFAVVRPLRFLGAAGFPDDGNTAKGIKKPITTGPWVLTEYKKDEEAVFTRNENYWGTKPKINKL
VVKVIPDGEARVLALQKGNIDLIWGEGVISLDSFVSLRDSGKFKTSISNPVATRTLIVNTAKGPFADLNVRLALQHAVNK
KALVDGVTNGVEKVADTVISSNFPYANVGLKPYDFNVETAKQLLDQAGWKLPAGKTIREKDGKPLEIELVYMGPDQIEKP
MAAAIQSDLKQVGISLKTTGLERELRNKRMESADFDLLFWGTYGAPYDPHNFVSTNYQPGYGIFESQQGLSMKDELHQQM
KDVLLTTDEKERQKIYTSIFTTLHDQAVYLPISYLTNIAVYPNYVKDLKFPAHRDETPFDTIDIQK
>lcl|NZ_CP026520.1_prot_WP_053228612.1_3332 [locus_tag=PC41400_RS16885] [protein=MFS transporter] [protein_id=WP_053228612.1] [location=complement(3704194..3704604)] [gbkey=CDS]
MDIVDIFLSGMMTSLTLPFLNILAGLLFAALPANLFIVVFLLRGGFFTILNNLVESHSMSMVEDSQRNQFAGMRSVFRSI
GAAIASYFAGFMLVNRDYQMPFLITGAVILIACGYFWFVVRPMFLKENKENATIID
>lcl|NZ_CP026520.1_prot_WP_156972775.1_3333 [locus_tag=PC41400_RS29860] [protein=hypothetical protein] [protein_id=WP_156972775.1] [location=complement(3704571..3704816)] [gbkey=CDS]
MNTPDSTNYQLQLWFAAIIMAMLALVRGRRLPTEKMSQTDLISDVGEATASSSMKGKNTWLSMSFLPSKSVWILSIFFYP
E
>lcl|NZ_CP026520.1_prot_WP_053228613.1_3334 [locus_tag=PC41400_RS16890] [protein=hypothetical protein] [protein_id=WP_053228613.1] [location=complement(3705053..3705268)] [gbkey=CDS]
MSKRKYPVGVRIFVLSEALFGIAVGIFSLVLNLHLLAAGVSETQIGRMNSVSTLLAGAVAVPGGWLAGKMG
>lcl|NZ_CP026520.1_prot_WP_042226181.1_3335 [locus_tag=PC41400_RS16895] [protein=ABC transporter substrate-binding protein] [protein_id=WP_042226181.1] [location=3705885..3706571] [gbkey=CDS]
MGVTPKQYARSQSRGTEQKNSSATEQYSRVIAIGYALGDLLSLGIKPVGADMTIIGKKVVFRDLLRGIADIGLLGEPGKI
KALEPDLIICNSFCMDWPEQLSNIAPTINIDRFIPTYKRILEVAELFGKETQARSWVRNHTHKLEEVWSKLSDQIGKDET
ASIFVWMEGTLYAMGMKGIGLTLYHPMAFKPSAIYCSALINKFTSSISEFNKYLSYFGMRLRSKNASI
>lcl|NZ_CP026520.1_prot_WP_042226183.1_3336 [locus_tag=PC41400_RS16900] [protein=AraC family transcriptional regulator] [protein_id=WP_042226183.1] [location=3707521..3708480] [gbkey=CDS]
MRIVTDSPLQVSNQSKGLLDEERSRLARMILALAPYDGAFKQRIPGLLISRFSRIDSDTVNDFNSPSLLFAVQGAKVITV
GQQVYTLDKLEMLLFPVALPVTMQTTNATPSEPFLGIRLDLDSRRISELVLKVYAHGLPPLGQRSAGYAANTGLSIMKAV
SRLLDCLSDPGDTEWIAPLVMDEILIRILRSPFGKQVAEMGFVDSGVQRVAKAIAWLRSNFSQQMKVTDLADLMHMSVSS
FNEHFKSVTSMSPLQYQKALRLHEARRLMFSDSRNASTASRQVGYISISQFSRDYSRFFGISPSRDIARWRQQEQVPHD
>lcl|NZ_CP026520.1_prot_WP_042226185.1_3337 [locus_tag=PC41400_RS16905] [protein=SDR family oxidoreductase] [protein_id=WP_042226185.1] [location=3708710..3709600] [gbkey=CDS]
MRVFVTGATGFIGSAIVRELIGAGHEVVGLARSEAAAAALTGVGVSVHRGSLDDLDSLRSGAAAADGVIHAAFVHDFRDY
AEANRQDSQVVKALGEALAGSDRPLVVTSVTALLTPGRLGTEMDTADPHSVARVASEETALAMASQGVRASVVRLPTSVH
GDGDKGLVPTLIDVARRTGVSAYVGDGTNRWPAVHRLDAAKLYRLALESASAGTRLHAIGDEGVALRDIAALIGRRLNVP
VVSKSPEEAVDHFNWLKHFVMMDNPASSALTQERFEWRPGQLTLLSDIDRQAYFGN
>lcl|NZ_CP026520.1_prot_WP_042226187.1_3338 [locus_tag=PC41400_RS16910] [protein=hypothetical protein] [protein_id=WP_042226187.1] [location=complement(3709854..3710258)] [gbkey=CDS]
MIYELKKSEYHRLKPILLSGFQFPEIYAVVNLVNSGWIVADDPVNPTSAFVWAEGLKGFFLIGCENNISFLEDLNHFIDH
ELNERLQRDVNGVEVAGMHQGWDDVIKQSYHCRNVKQSIQLIYKWDDHQAVEAI
>lcl|NZ_CP026520.1_prot_PC41400_RS30360_3339 [locus_tag=PC41400_RS30360] [protein=cyclic peptide export ABC transporter] [pseudo=true] [partial=5'] [location=complement(3710287..>3710557)] [gbkey=CDS]
FSTTELSTGQRKRLALLISYLEDRPFCLFDEWASRPRSRIS*VLLLQYTSRVKSAGKMRNCNFA**SLFSHCGQNNQNGF
GPSCEHGNEV
>lcl|NZ_CP026520.1_prot_WP_042226193.1_3340 [locus_tag=PC41400_RS16920] [protein=extradiol dioxygenase] [protein_id=WP_042226193.1] [location=complement(3710515..3710976)] [gbkey=CDS]
MALTSAFTSFNLPVKNVEQSKAFFTGIGFELNPQFPENENSVAIVIGDNLQVMLINQAFFNTLTEKETVDTSKYAQMTIA
LAFESREKVDEIVNTAVSLDAKLHAEPEDHGFMYHWGFVDLDGHLWAINYMNIDAAQITDSAQPNYQRDKGSV
>lcl|NZ_CP026520.1_prot_WP_042226196.1_3341 [locus_tag=PC41400_RS16925] [protein=choline esterase] [protein_id=WP_042226196.1] [location=complement(3711255..3712397)] [gbkey=CDS]
MKNYDAFPEPIGSYTVGRTQMDLEYTASDHSKRELTAFVYYPSDSSEGKTTSTYMFPEVYEMFNEQPLVTPLKDAFSIDI
KTQCYDDLALSGKEKRYPVLFYVCGGGGSPEWGTVICTDLASMGYVVVSIGHQNSTMYKRKDGRLFNVSKDFSDVIMAFS
EDPEMLVLAGKMEMRPDDETAIEMCHNVLTLPIVSKLTEYSELQAEDVRYVADYLYKLDSGELNSNFKGRLLLDIGMGIV
GHSYGGLTTAIVCRDDDRFACGIGLDSGAFGLQGSDLKKPFLLLFCEPNYNMNAIIGANNSMETYYFSVDRVAHLDYCDI
VFTSVNEEIRGERDAMEMRNLVTDYTKNFFDHYILQKAASVESLAYDGVDLIKKYQQQVT
>lcl|NZ_CP026520.1_prot_WP_042226198.1_3342 [locus_tag=PC41400_RS16930] [protein=TetR/AcrR family transcriptional regulator] [protein_id=WP_042226198.1] [location=complement(3712510..3713169)] [gbkey=CDS]
MPKNSFFRLDEARREEISNSAMHLFVDNLYEDITMKMVLDCLSMHPGTFYRYFEDKDDLYCLLIRNVTQKRAAYFNSSNE
DSLFQFFLNGLFGNVNGIVTEPLNELEIKLTKTFLYIPENILLKVYLNVLKGESSPLIKGILRRMRVDGYLRPDIDDDLI
SFMFESMQFNLVMFFREFDIKDSKLQHKISKYFADFMGHGLLEDHKYSEMVSDLKKAKE
>lcl|NZ_CP026520.1_prot_WP_042226203.1_3343 [locus_tag=PC41400_RS16940] [protein=GntR family transcriptional regulator] [protein_id=WP_042226203.1] [location=complement(3714447..3715160)] [gbkey=CDS]
MNLSRNHRPLYLQIKNILKDRILHGVYPIKANIPSEPQLEKEFEVSKITIRNAIKELCQEGYLEKGSGKRTKVIRNTATS
KLSTWKRFTELLVEEGNQVQKQLLQVEVIMTEEGTEPHQLFGAQCVRIERLYSLNNVPYTHYTHYLTMRIGEIDQWELNA
QSLYEWLEEQEITLAKYRDHFAVAFASPLVEEKLRVQEKTPLLKRLRYSYDETDEVIEYSEGFYNTGMHSYIVNYNV
>lcl|NZ_CP026520.1_prot_WP_042226206.1_3344 [locus_tag=PC41400_RS16945] [protein=amidohydrolase/deacetylase family metallohydrolase] [protein_id=WP_042226206.1] [location=complement(3715309..3716421)] [gbkey=CDS]
MYEPKQESFVLRNVKRVNGESTDVVIEEGKITEVTKAGSGQGKQVKDCSGIYISSGWIDMHVHAFPEFEPYGDEIDEIGI
KLGVTTIVDAGSCGADRIAELAASKQHAQTNLFAFLNISRVGLQRIDELSNLEWIDTLKVIQAVKEHKDFIVGLKARISK
SVVGESGIEPLRIARALSDKTLLPLMVHIGSAPPSIEEIVPLLEKGDIITHYLNGKPNNLFDDQGNPLPVLADAIERGVS
LDVGHGSASFSLRIAEAAKQHGIGLNTISTDIYRRNRLNGPVYSMANVLSKFLYLGYSLEEVIAAVTDHAANWLGKPELG
RIQVGDVANLTLFTLRNDPITLVDSEGVKWTADRSIEAKGVVVNGVYMEC
>lcl|NZ_CP026520.1_prot_WP_042226207.1_3345 [locus_tag=PC41400_RS16950] [protein=hypothetical protein] [protein_id=WP_042226207.1] [location=complement(3716925..3717140)] [gbkey=CDS]
MALNSWEKAIQLDPEWLDSRLARASLLERDQRLQEAAQEWRLVIQWLKNRGFFTDRPNAELQRIEKKINNN
>lcl|NZ_CP026520.1_prot_WP_129112501.1_3346 [locus_tag=PC41400_RS16960] [protein=DUF3221 domain-containing protein] [protein_id=WP_129112501.1] [location=complement(3717690..3718112)] [gbkey=CDS]
MKTKVDESIVGYVVKKEADRFLIVSSVADSNNEYSATWVLSSEEVKIGQLVEVFIEGGEIKASDPGQATSKKIILREIKV
GESSNQEASKVLENVLLQHTELEVPVVKELAYSKDSKEWSVTLFDKKDANKIEISIKLKD
>lcl|NZ_CP026520.1_prot_WP_042226212.1_3347 [locus_tag=PC41400_RS16965] [protein=hypothetical protein] [protein_id=WP_042226212.1] [location=complement(3718257..3719408)] [gbkey=CDS]
MRKQLKITALSALLSLTLISSSFLNVENGTANASALTPEEVSVQHQNLKVAHDIIGNYEDYSTLYIDDNNDIVLSISKAD
TKTEKIEKDLKEKLPKVKFSKSKYSTKELKDVSFNLYKKFPELKKESANVVSTAIDEELNKVVVVATELPEKTREKLIKE
YGDMLEIKVDKKYTGEIETSRTSNNAVLGGGIAANNSSFTITATAKKGSDRYLVTVGHALTGDGTTVIKQNTTNVGVDWA
KAVSHDIGLIKVTATGRTISNKIIRNSTDYDYKYTSTGSVTQGSTYCKSGIKTGYQCSKVISTSYNGLYQDTIQLENPNW
TFQDNGDSGSPLFSSSSLSYVLYGIMSTKSTASGTDAFATATKISYLNHYWSDYSLYTSDTAY
>lcl|NZ_CP026520.1_prot_WP_053228614.1_3348 [locus_tag=PC41400_RS16970] [protein=acyl carrier protein] [protein_id=WP_053228614.1] [location=complement(3719817..3720080)] [gbkey=CDS]
MSNNQTSIEDIKIIKLIEDKVKAILKHDRDITLDEDLRKLGLDSMKSFEMVVQLEDEYNIEFSGDENLLERFSTIQKIKD
SVDSKIK
>lcl|NZ_CP026520.1_prot_WP_042226213.1_3349 [locus_tag=PC41400_RS16975] [protein=hypothetical protein] [protein_id=WP_042226213.1] [location=complement(3720210..3721208)] [gbkey=CDS]
MNSSFVGTDVKPFNEFWMNCKYNALFSIASYYDPNNKYASYVNHYSFEHTNNYKDPSFKLPIIKHAEFSEMLPNFTYSIE
NCHFENEENFLEELKQMIRQGQLVFLSVNLYSWIPGSFCYQRYHWSHFSLIKGYDEVNNCYLVLDDDMNGYNEFVVSEDT
VVKCFRSSDYFTDRENWPYPIVKLQLHQPLLPYEVTLQTVQDHARRLVLDIEQFTIDDLWKFDEESELLSNHLDYSVIGV
EYFYNRHIANVKLFKHLHKHKKINDELFETMLHHAKNLENGWQTIRNNLMKVKAKLKKSRNVPRDSIIAEAKLLFENEKL
MWQLLVEAKSEN
>lcl|NZ_CP026520.1_prot_WP_042226216.1_3350 [locus_tag=PC41400_RS16985] [protein=2-dehydropantoate 2-reductase N-terminal domain-containing protein] [protein_id=WP_042226216.1] [location=complement(3721976..3722893)] [gbkey=CDS]
MSAKQDRILIFGAGVIGSMYAIKLIEAGFDVSLFARSNRFKSLRENGLQYKEKGTVKSIQVNVIDTLENDDVYDFIFVTV
RYDRSESALLALKDNQSKNIVTMTNNSIGFSSWLDIVGDRLLPALPGFGGQIKDGVLHARFLPKIIMATAFGEINGVMTE
RIENLAKLFKTAKLPYVVKKDMQGYLITHSVFDIAMMSFLHSENKIIDKKTARKTARKITVTLKAYLRAIQKAGVSIDPP
MLKMVLKFPNLFLDLFFTTWLRTKMVRDMMLPDYANNASNEVVQLSNDLMKFLSQNDIKSELHVQ
>lcl|NZ_CP026520.1_prot_WP_042226219.1_3351 [locus_tag=PC41400_RS16990] [protein=TetR/AcrR family transcriptional regulator] [protein_id=WP_042226219.1] [location=3723042..3723593] [gbkey=CDS]
MKKQPQITEKTRQKFIEVFCELYSQKPIEKISVQEITNKSGYNRSTFYQYFTDIYELLDSVENDLLNEIKKELANKELSM
HTVQDTLYCLDKREHLLVLNALLGDYGSSRFLKRLKKEITLDQLELNVPQNHSLTPYIIEFYLTTSLSLFRLWIQRQKDL
SSEEFFKLVENLYSRGITSYFKE
>lcl|NZ_CP026520.1_prot_WP_042226222.1_3352 [locus_tag=PC41400_RS16995] [protein=FAD-dependent oxidoreductase] [protein_id=WP_042226222.1] [location=complement(3724466..3726103)] [gbkey=CDS]
MMSKKITNVPVLIVGGGLAGLSAALFLSRHSVPYVLIERHQKTNIHPRARGLNFRTMELYRELGLDKAIEEIGAQLAKGK
GWYTAHTLREAEFDPEKMDIPKILPESVKSYFATLKEISPASSVRATQNLVEPILLEAARERGGDLRFHTELTAFEQNED
HVRATVLNRTTGVEETIVADYMIAADGAKSPVRQMLNIKMTEMNSHGHIMNIYFEADLEHFVKGREFSACNITRTGAAGV
LLAINNRDRWCYHVSYDFKGGETAAHFSEERCREVIQKAIDLPDLQVKILSVLPWEAAEKVAEQFQQDRVFLIGDAVHLM
PPTGGYGANTGVQDAHNLAWKLAAAVHKAASPELLKTYEAERRPVAQMTVSEAGRVADTGVFSAMKKQTEKRTSPFDDLI
VSVGYHYESKAILKQDTAPSSLDHFELKGRLGTRAPHIWGEYQGRECSTLDLLGLGFVLFVGEKAEGWEDAAKYVSGQFG
VKINTYSVGQNGDFIDKDHGWQNAYQAKEKSIVLVRPDGFVCWRCEEHIEQPTLTLKEVFAQLLG
>lcl|NZ_CP026520.1_prot_WP_042226225.1_3353 [locus_tag=PC41400_RS17000] [protein=TetR family transcriptional regulator] [protein_id=WP_042226225.1] [location=3726318..3726959] [gbkey=CDS]
MEDHLKLSERQQEIVEAALELLAEKGYDELSIRDLGKKLDVKAPAIYWHFKNKAMIVDYMAEHILQKKMGNFVPREREQS
WQDWLIEHITLFRKAMLSYPDGGRVIAGAHPFPNGTLAKFLEDSFVSLRSTGMDIRTARIVMLTVIRYTFGCVIEEQADE
HNWETLPERAINSGELTNIMEAIQVGGTDDENFMTGLKLIIAGGNEAMHAPAP
>lcl|NZ_CP026520.1_prot_WP_042226228.1_3354 [locus_tag=PC41400_RS17005] [protein=SRPBCC domain-containing protein] [protein_id=WP_042226228.1] [location=complement(3727129..3727557)] [gbkey=CDS]
MANIEHLQTVNVPASTVYEALTTTKGLSEIWTNELVVNDQIGYINEFRFGGKGLTKMRVDELVTDKKILWQCVDSDPEWI
GTTISFDMEEKSGKTTIIFRQMNWEEVTAFYRACNYNWAIFLYSLKQYCEEGEGLPYHKRKF
>lcl|NZ_CP026520.1_prot_WP_129112502.1_3355 [locus_tag=PC41400_RS17010] [protein=NAD(P)-dependent oxidoreductase] [protein_id=WP_129112502.1] [location=3727662..3728228] [gbkey=CDS]
MKKCIAEGGFNYLDGTIIVPAIGQPEASILYSGSENAYLTHQRTLMSLGGTASYLGTDPGSAAAYDVALLDIFWTSMTGY
IQAFALASVENIRATDLVPYARNIIGMMPDIMAEFANQVDNGHYPGIDSNLISTEVVMDNVIHASKARGIDVGVPDAAKS
IVRQAIDLGYGKQDFSRLAELFRNRPRD
>lcl|NZ_CP026520.1_prot_WP_042226231.1_3356 [locus_tag=PC41400_RS17015] [protein=MerR family transcriptional regulator] [protein_id=WP_042226231.1] [location=3728268..3728630] [gbkey=CDS]
MKISQLSKTTGASIRSIRHYEETQLLTASRLKNGYREFDEAAIERIKTIKAYLNLGLTTKQIEETLKCTDGYYEDEAEQV
CREMIHIYEEMLLEVNRKMNALAVVKQELEEQSRQMKDSL
>lcl|NZ_CP026520.1_prot_WP_042226233.1_3357 [locus_tag=PC41400_RS17020] [protein=dienelactone hydrolase family protein] [protein_id=WP_042226233.1] [location=complement(3728637..3729944)] [gbkey=CDS]
MQILILVVTLVFELAITVYSIATKQSRSKIKSWTRIAMFIGFVMLILGKVVVWKYTWGLFAGLLFILAFKDTVALLRKQT
HTPRYKVSSTIWKFLLLALTVIVTLVPVLLFPQHRLPQVTGPYAVATATYSYVDKNRIEKFTDQEDNRFVNVKFWYPGQA
DGTYPLLVFSHGAFGIKTSNTSTFTELASHGYVVVSIDHPYHSFYTVSEDGKVVTINPGYMQEVNNANKEGVYSLGEFFE
LTQKWMKLRTDDMDFVMDTILEQAEQKKDSVYERIDTQKIGVFGHSVGGTASVALSRERNDIDAVVNIDAPFFSELVYDK
VTNALIAKSEAYTIPLLNIYSDDVWIQLDSSSSYIANRISNKNFKGAYNVHFKGAKHLSLTDLPLFSPILANLLQEGRKA
DIDKYYAIETQNELILRFFDYALKNQGQFAPKATY
>lcl|NZ_CP026520.1_prot_WP_241688676.1_3358 [locus_tag=PC41400_RS17025] [protein=LytTR family DNA-binding domain-containing protein] [protein_id=WP_241688676.1] [location=complement(3729973..3730251)] [gbkey=CDS]
MSPSRRQMHELVNKIKTETLIVLGYHEDKISRIKLSDIYYFEAVNGKVFAYGKNTVHEVKQKLYELEELCKEKHCFRASK
STIARPFIMMNS
>lcl|NZ_CP026520.1_prot_WP_241688904.1_3359 [locus_tag=PC41400_RS17030] [protein=metalloregulator ArsR/SmtB family transcription factor] [protein_id=WP_241688904.1] [location=complement(3730833..3731120)] [gbkey=CDS]
MFKALSNSSRLQILFWLKDPETHFTPYEGVDMRKVGVCVTQVMEKLKMTQSTASQYLSILQKADLITVERIGKYTYFKRN
EETLREIGAFFNNEI
>lcl|NZ_CP026520.1_prot_WP_042226237.1_3360 [locus_tag=PC41400_RS17035] [protein=nuclear transport factor 2 family protein] [protein_id=WP_042226237.1] [location=complement(3731626..3732405)] [gbkey=CDS]
MTIKSQNELVHKAIAVLESFESGNPEAITAYVHPKQYIQHNQGLSDGHAAMLGALGHLKEIGTKVSVKRALVDGDYVALH
SVYEFFGSKIIMDIFRFENGLIVEHWDNMQEMVEKTPSHHTMIDGPVTIKDIDKTDTNKEFVKSYVENILLGKNPGLLAA
YFDGDNYIQHSPHIADGLTGLQAALQELKKKNIEFEYTHVHQVIGQGDFVLTVSEGLFDNQPTAFYDMFRVENEKIAEHW
DVIEAILPAEKRKNSNSRF
>lcl|NZ_CP026520.1_prot_WP_042226239.1_3361 [locus_tag=PC41400_RS17040] [protein=TetR/AcrR family transcriptional regulator] [protein_id=WP_042226239.1] [location=3732599..3733222] [gbkey=CDS]
MNKKIDLRILRTKQSIRKAFYELIQEKGYEAITIQDIADRAMINRNTFYLHYPNKPDLLDTCMNELLSELKDAVVLCPIG
MNPFSISILETVMQTVLEHISLNTTFYYAMLIEEKRIYQFRAKMENIIKDKLNEGWNPDQVNAPLAISKDLLLEYLVSAF
MGIVIWWIKNDQHLPANEVSSQFSRIVAYGHLRAAGIPVEDEYSPER
>lcl|NZ_CP026520.1_prot_WP_042226241.1_3362 [locus_tag=PC41400_RS17045] [protein=dienelactone hydrolase] [protein_id=WP_042226241.1] [location=complement(3733339..3734841)] [gbkey=CDS]
MRLFELLFFLSNIGLFALTVLLKKGRRRIPVFVASGIASFLLVIHWTVEGYRVQLFFSYCITIIFLAISGYRYFKKTGLK
KIPRFMLGSAYTAIAIMLVVTAGLMYAFPVFKLAEPTGEFKVGTQTFHFVDPNREEIFDKTRKGKRELMVQVWYPAQAGT
GKYAPVIPDTRILRYMAANYGLPGFTLQHLKYVSSHAYSGAELSSAQTSFPLILANPGFGSSRFLHTSQAENLASHGYIV
AVIDHTYNTFATEFPDGRITTSTTNDLFSPDHDYRTESGNRDKLGKVLTDDVAFALDQFELIQTGQIPSHLKGRIDLGHV
GVFGHSIGGATAYDASYDPRISVGIDLDGGLYRLRDREGLRKPFLFMNSESEFERLKMVMDNHAYTEEELKRMGNTREWM
DQVTEDKKLELERMRETVSAGGQFLYIDNTEHLNFTDVQFISPIFKMLGVTGKIGPERANSVINAYMLDFFDMYLKNQGG
NLMKGPDSRFPEVKFVTSLL
>lcl|NZ_CP026520.1_prot_WP_042226243.1_3363 [locus_tag=PC41400_RS17050] [protein=bis-aminopropyl spermidine synthase family protein] [protein_id=WP_042226243.1] [location=complement(3735108..3736310)] [gbkey=CDS]
MKTYIEQASQNTRLQEGTQVIEQLLLECYLNPGISTKGLARKTLLPTPVTAAIKRELIKAGALVQDRGVRCTADGLAWIE
REWGYGGLDHSLYYDLLDETTWIESLKDILVILEELFALRPSVDVQIDQSKCTPETSLRRAILCLKQHALIGKKILCVGD
DDLVSVSIGLLLQRLFPDGGHTVTHVDVLDIDERFLRYIETIAKERGLPIGCHQLDLREPLPENLHGQFDCFFTDPPYTL
QGMGLFVSRGIQALKREKGLPIFLSFAHKSPSFMLAMQREFVRMGLTVSANFPQFNAYEGAEMIANRSQMFILRRTDQTK
PEYSETFTDALYTGEVKQTLRTYRCKQCSREVYVGIHGEFATIEQLKNEGCTGCGNDTFDMVAKKQVSQERNDKDDHTAG
>lcl|NZ_CP026520.1_prot_WP_129112569.1_3364 [locus_tag=PC41400_RS17055] [protein=alpha/beta hydrolase] [protein_id=WP_129112569.1] [location=complement(3736860..3737636)] [gbkey=CDS]
MTTADINGYSMHYIDQGKGTAILFIHPPVLTSLNFTYQIQGLSPHFRTIAFDIRGHGKSQPSKQTVTYPLIVEDIKQLMD
RLSIDKFFLCGYSTGGSIVLEFLLTYPDRAWGGIVIGGMSEVNERWLRNKISLGVAFSKIGALGTIAFSNALTQAKDLSL
FRELFNDAKKGNAKNAEQYYKNSLKYNCTSQLQEIYLPVLLVYGEKDKHFHPYAKILHQRLPKSELVFIKKVDHRIPTKA
SEPLNELIKQFVHRFGHS
>lcl|NZ_CP026520.1_prot_PC41400_RS17060_3365 [locus_tag=PC41400_RS17060] [protein=TetM/TetW/TetO/TetS family tetracycline resistance ribosomal protection protein] [pseudo=true] [location=complement(3737916..3739896)] [gbkey=CDS]
MHTMNIGILAHVDAGKTSLAERILFDTHVINELGKVDQGNTQTDAMELERKRGITIKASVVSFVAHGLKINLIDTPGHAD
FIAEVERSLSVLDGVILVVSAVEGVQAQTKILFSVLNKMGLPTILFVNKIDLTGAHFDATIMRIRDKLTPNAISLCRVEH
AGTKMASVAEHRFDWNTDPAFLEQCIELVAEYDEQLLESYVNGEGISEEQVKKKFGKQVRQSECISGLFRFRRDGRGRIP
FALRCDRVVSGERACGERAVVGCRVQAREGAVRRKGSLCQTVLWQHEHEGTSKAAAKKQAGRI*NAHLQNQKTACPNKRK
VSSRCQSNLRGFGQSMGTRECADRGCRRRAVRFDQALQLRRSANGDANRTGRTKQSASAVPGAYWHVGGRSVNPSGER*F
SSRPVYPHFRRSAKRSN*GGLAGGL*IRCPVLGDENHLPGKSGRNGAGGGDYHGGQ*SLFCDCRTKSRARFARKRDRLPS
GSRVGLAAACLP*SH*GDGLHYVETRLVRLGSHRYCRLLDPNRLF*PDIGSG*F*KAGTPCIDGGAGASGYPCI*AYSCV
RVNSPRRDIEQSGIQTIRRTSGCTRNGYA*RHVLYNRHHPRSRHRRLQEKLAYIYPG*RRHAGRAGRIHPDKRRLADKRA
NRLQSVESERIFASCSARLL
>lcl|NZ_CP026520.1_prot_PC41400_RS30365_3366 [locus_tag=PC41400_RS30365] [protein=tetracycline resistance MFS efflux pump] [pseudo=true] [partial=5'] [location=complement(3740073..>3740537)] [gbkey=CDS]
SVLFLGYIGGLLVDRKGSLFVLTIGVAFLSISYLIAALFIGATPWLMTIVLVFVFGGLSFTKTVISTIVSGSLKQKETGA
GMSLLNFTSFLSEGMGIAVVGGLLSIGLLDRRLLPMNADPSTYLYSNMLFLFAGIIAISWLVTMKVYKRSQRDF
>lcl|NZ_CP026520.1_prot_WP_241688677.1_3367 [locus_tag=PC41400_RS30370] [protein=class I SAM-dependent methyltransferase] [protein_id=WP_241688677.1] [location=complement(3740497..3740841)] [gbkey=CDS]
MFDADSFDLIVHPVSNLYVPEIQPVWQEAFRVLKPGGSLISGFMNPVFYLFDWDLQEEGVLQVKHSIPYLELDYSTEEEI
AKNRIPLEFGHTLEQQLQGQIECPISRLHWGITC
>lcl|NZ_CP026520.1_prot_WP_042226322.1_3368 [locus_tag=PC41400_RS17075] [protein=YfbM family protein] [protein_id=WP_042226322.1] [location=complement(3741068..3741556)] [gbkey=CDS]
MGMRGYYFALDDNLVQQIAAGDIALKSLKIDNYPGLDIDRSWEAIHYLLCGDISDGEPPLGYVVPLTSDKGIDFGSFGAF
SLRAEQVAEALQAMSELDEAQLRLRYDFPAMVKEEVYPLEAGTVSDEDEDALFAYMLQHFNEIRRFYSQTAAEGKGLIFY
IF
>lcl|NZ_CP026520.1_prot_WP_129112504.1_3369 [locus_tag=PC41400_RS17080] [protein=hypothetical protein] [protein_id=WP_129112504.1] [location=complement(3741853..3742278)] [gbkey=CDS]
MSHLLRLNQMKNKNYYELFPGVLTKGYSNAESVYIEDDLFVFLRPAIKNGYEPFESFNRFEIPKENWVQIISGFVIIREY
LYNHQTTELAEYIQPHFKTQTESFLKKYLKNRVLIDEIINDLITWIEEQLITNECITLIGL
>lcl|NZ_CP026520.1_prot_WP_042227365.1_3370 [locus_tag=PC41400_RS17085] [protein=D-serine ammonia-lyase] [protein_id=WP_042227365.1] [location=complement(3742411..3743772)] [gbkey=CDS]
MRRIDQVKEFEFSELPLLNKLISTKEILWLNPKLEKFETGITKSPLTQDDVRDAEERLKRFAPYIAKVFPETKPMNGIIE
SPLKRIPSMKQFLEQNYQQPILGELLLKCDSHLPISGSIKARGGIYEVLKHAEELALQHQLLTIQDDYSILDSARFRTFF
SQYSIAVGSTGNLGLSIGIMSAKLGFNVTVHMSADAKQWKKDLLRNKNVNVIEYESDYSKAVEEGRLQADGDPSCYFVDD
ENSHDLFLGYAVAASRLKKQLEELEITIDEHHPLFVYLPCGVGGGPGGIAFGLKLLYQDHVHCFFAEPTHSPCMLLGLMT
GLHDKISVQDIGLDNVTDADGLAVGRPSGFVGKTIEPFLSGNYTVSDEHLYTLLKELVDTEGIHLEPSALAGMIGPIKLS
QEGTDYLLKHNLTEKMSKGAHIIWGTGGNMVPEDMMRQYYQKGLNLTRTKKNP
>lcl|NZ_CP026520.1_prot_WP_042226326.1_3371 [locus_tag=PC41400_RS17090] [protein=XRE family transcriptional regulator] [protein_id=WP_042226326.1] [location=3743916..3744461] [gbkey=CDS]
MENIDIGKKVEKYRKAKDFSIRQLAKLAEITPSMLSQIERGLANPSIQTIKVLAKSLEVPTFSFFMEETNTVDLVVRSTN
RKKMVMDNISYELVSPDFTGNLATAIMNLPPNTASSEKQMEHKGEEVAFVLEGKIKVILNDEEYVLEPGDSVKIPAQMKH
KWENNFNEHAAILFSVTPPAF
>lcl|NZ_CP026520.1_prot_WP_042226329.1_3372 [locus_tag=PC41400_RS17095] [protein=hypothetical protein] [protein_id=WP_042226329.1] [location=complement(3744591..3745427)] [gbkey=CDS]
MKKICSMLGLLIAVILIFIPQDASARSFFEHSTTTVPAGQTMDNLYVLGGDADILDHVTGTVVVVNGNLHLASTAKVDGA
IVVIGGRITQDEGAQTGDDILNFTLDTATQNSLLIGGGLVAGVWAVQLTCSLLLVVIPVLTYLIGGRKAAGWIKRQPDLS
WKRQLYLGGLSFIILAAFCTLCIVTIIGIPLLLLAILLLMIAYIVGITSLSYLVGGFLQEKWVPNEWVRLIIGASLIAAI
ANIPVLGWLMILFLTLFSLGTFTKCLSSLRRRKPREIR
>lcl|NZ_CP026520.1_prot_WP_042226332.1_3373 [locus_tag=PC41400_RS17100] [protein=zf-HC2 domain-containing protein] [protein_id=WP_042226332.1] [location=complement(3745441..3745920)] [gbkey=CDS]
MCPEEQLSAYMDEELNEEERLEIESHLAHCQSCRLLLNEFMELKQSFSAEMFALAEPAGFENRVMQAVAHESKIRNLGKL
WLLVPLLAAVVVILLTIAAWPVAVHFMNGALVIGKALLFTIPHAVSSMPVVAGVTVALSLTVLVVSLLSLRRLLRSATV
>lcl|NZ_CP026520.1_prot_WP_042226335.1_3374 [locus_tag=PC41400_RS17105] [protein=RNA polymerase sigma factor] [protein_id=WP_042226335.1] [location=complement(3745922..3746455)] [gbkey=CDS]
MVDEELSKIIERARRGNEEAFSELVTRFKGHVFRHTYGLLGDRGEAEDAAQEIFMKVYFALGKLDDVHAFYSWLMRIVSN
LCKDKLKQRARSVEIGEMSNETVPDFSARDPNLKVSLQEALSHLSVDHREILLLHDVQGFRYEEIAEQTGIPIGTVKSRL
FSARMALRKIWNRGEGD
>lcl|NZ_CP026520.1_prot_WP_053228618.1_3375 [locus_tag=PC41400_RS17110] [protein=undecaprenyl-diphosphatase] [protein_id=WP_053228618.1] [location=complement(3746792..3747454)] [gbkey=CDS]
MLSIDYQLFQIINGMVGHYSSLDAIMKFFSNDGEYVFILGIIVYWFTRTELNRRMVMEALVSACIAFGITAVLGSLFYRN
RPFVDHHVQLLIQHAPNASFPSDHATAAFVVATAIWCTRRKDGWWWLALAAGITVSRVWTGVHYPLDVIAGCLLGVGTAI
GIHSCIAKSKWIQICLTKVIRVYERFEVSVWPKHTSSSAFISKSLQYEKKQEEVCDKKTN
>lcl|NZ_CP026520.1_prot_WP_042226336.1_3376 [locus_tag=PC41400_RS17115] [protein=YheC/YheD family protein] [protein_id=WP_042226336.1] [location=3747848..3748603] [gbkey=CDS]
MEYKSSIIQSKWKKTQWLLKDSYFRKYVPHTLPFNKKNLSAMLDNYSSVFFKPTNGSGGTNIIRIRKTDRGYLSQMNAAK
TSYSTTNQLYRDLNRFAGSRSYLLQKGINLAQSNGKPFDIRVMVQKNRQGDRVSTALFTKIGNPNKVTTNYTQGGTVRTF
SETMNGTDFSPTFTPQMESKLKRLGVAVGKNFDRHHKGFQELGLDVALDSKGRPWILEVNTRPQIYPLKALKNKNLYDKI
LSYGKHYGRFK
>lcl|NZ_CP026520.1_prot_WP_129112505.1_3377 [locus_tag=PC41400_RS17120] [protein=hypothetical protein] [protein_id=WP_129112505.1] [location=complement(3748629..3748811)] [gbkey=CDS]
MRLFFVKGYGTLKAADLALSFFGRKISSSKRQGHVSYKASGEVALSGHYDGEHDHDDDED
>lcl|NZ_CP026520.1_prot_WP_042226337.1_3378 [locus_tag=PC41400_RS17125] [protein=MarR family transcriptional regulator] [protein_id=WP_042226337.1] [location=complement(3748910..3749401)] [gbkey=CDS]
MNSLEITKRMVYDQVLNYVHLSEQMIESEQGNIFQYAKELQLQSIPRHLTSVHVIDCIGNNKQINNTALAERMNLSKANI
TKISTRLLEEGFIKRSQLNDNKKEIYFSLTAKGRQIFELHEQLHKLEEERFMQLLEPFAEAELHVILRFLQNVSSHWANK
ETE
>lcl|NZ_CP026520.1_prot_WP_206735004.1_3379 [locus_tag=PC41400_RS17130] [protein=MDR family MFS transporter] [protein_id=WP_206735004.1] [location=complement(3749503..3750885)] [gbkey=CDS]
MNSNTINIALPVLQKEFASDLSVVQWTLTGFMLALGTFAPTAGYFGERFSYKRLYMFALLGMTVASVLCALSWNAPMLIA
FRIIQGAFCGIIVPASMTMIFQTIPRHQQSAAMSIWIAASIVAPAIGPSFAGWLIQHGSWKWLFWTNVPLGLLAIVFAFF
FIPYYRLKVPKGFDLWGLLTVTAASSLLLITLSQGSGWGWGSASTLICLGSGLLALALFLWREMSIESPLLQLKVFLNRR
FTSTLLLLCIVMINFFSGMLLVPVFLQNVQGHSPLDTGIILLPSTLAMAVMSLVVGKLYKVVGPKPLLLVGVLLLIVGNL
PLAWLQIDSSSAYTLLWMTIRSIGLALVLMPATTAGMEEISHEYIGHASSIQNWLRNVFVSFGVALFTTLLSSRSADHAA
KLTNEDAGPQLAHLSTTMGINDLNILSTLIVVLALPFVFAIPSKRKKERAVDGALSDPVR
>lcl|NZ_CP026520.1_prot_WP_042226341.1_3380 [gene=sufB] [locus_tag=PC41400_RS17135] [protein=Fe-S cluster assembly protein SufB] [protein_id=WP_042226341.1] [location=complement(3751097..3752482)] [gbkey=CDS]
MVELGEYQYGFKDAHQAVFQSGKGLTREIVATISEMKGEPGWMLDFRLKSLELFQNMPMPAWGGDLRELDFNDIQYYVKP
SEKQGKTWEEVPIEIKQTFDKLGIPEAEQKFLAGVSAQYESEVVYHSIQKELEEQGVIFTDTDTALREHPDLLRQYFGTI
IPPSDNKFAALNSAVWSGGSFVYVPKGIKCEVPLQAYFRINSENMGQFERTLIITDEDSFVHYVEGCTAPIYSTNSLHSA
VVEIICKRNSRARYTTIQNWAPNIYNMVTKRAVAEENATMEWVDGNIGSKLTMKYPSVLLKGRGAKGSVLSIAVAGKGQH
QDAGAKMIHLAPDTTSTIISKSISKHGGKVTYRGLASFGRNASGAKSNIKCDTLILDNESTSDTIPYNEIMNDNVTLEHE
ATVSKVSEDQLFYLMSRGLSEDEATQMIVMGFIEPFTKELPMEYAVEMNRLIKLEMEGSIG
>lcl|NZ_CP026520.1_prot_WP_042226343.1_3381 [gene=sufD] [locus_tag=PC41400_RS17140] [protein=Fe-S cluster assembly protein SufD] [protein_id=WP_042226343.1] [location=complement(3752510..3753826)] [gbkey=CDS]
MTMNRAENRVSGQLHYEAEPDWLTRLRKKGLNALSSLQLPKLEKTNLRRWPLRQQGICRKPELMMSLDELPQSLRKWVYS
DDMLVQRNSGPLIKHLTERLQKQGVILTDLETAARQYPELVEPYFMQAVHAEENLLTAQHAAHWSGGAFIYVPAGVHVET
PLQTLFYVDESDTLFMPHVLVVAEERSSVTLVEHTMSSLDKCAAVNQAVMEVFVGEGAAVNVVSLHQLERDSIDLTYRRA
IVEQDGRVNWLIGEMNWGKSMSDTASILKGEGAGSDIKSILVGSGSQTMNITARTVHIGKNSPSDMVIRAVMREQSTAIL
NGITKIEKGASGANGQQTEKVLMLHPHARGDANPILLIDEDDVKAGHAASVGQVNPEQLYYMMSRGISRVEAEQLIVYGF
LAPVVAEVPLEGVAAQMQALVEQKLGQIRETDQASLSE
>lcl|NZ_CP026520.1_prot_WP_042226344.1_3382 [gene=sufC] [locus_tag=PC41400_RS17145] [protein=Fe-S cluster assembly ATPase SufC] [protein_id=WP_042226344.1] [location=complement(3753843..3754670)] [gbkey=CDS]
MTVSLRIEQLRAAVEGKEILKGLSLTVNGGEIHVVMGPNGTGKSTLASALMGHPKYEVTEGMVTLNGEDLLEMEVDERAR
AGLFLAMQYPSEITGVTNSDFMRSAINAKREEGQEISFIKFVRMMESSMQTLEMNPEFMHRYLNEGFSGGEKKRNEILQM
MMLEPSMVILDEIDSGLDIDALRIVAEGVNALRSADRGFLIITHYQRLLNYIKPDFVHVMMQGTIVKSGGPELAERLEAD
GYEWIKEELGITEEAAAPDEEKSFQIPMQTKSPRI
>lcl|NZ_CP026520.1_prot_WP_042226346.1_3383 [locus_tag=PC41400_RS17150] [protein=helix-turn-helix domain-containing protein] [protein_id=WP_042226346.1] [location=3755110..3755505] [gbkey=CDS]
MITFRDKTYRCTSEIILGLVSGKWKISILNHLSKGPVRFNQLQRLLPDATQRMLTMQLRMLESDGLIIRKVYPVSPPKVE
YGLSELGEQLAPMLTMLCDFGTTYIESYPDEVSYTDQKHTLEVASKNLRGT
>lcl|NZ_CP026520.1_prot_WP_042226347.1_3384 [locus_tag=PC41400_RS17155] [protein=hypothetical protein] [protein_id=WP_042226347.1] [location=complement(3755583..3757007)] [gbkey=CDS]
MQEYQLTLKDKRIVWGKAIDIESLIGQYPSDSIRQGANEALDWNLPAGIYRAKEVVLELDKMLEAILVQLGEPVNGDPTL
LLDSLQANLAISGRLSSLPLGPLALEDKAGAEITAQAQRIGEQLVSWAREMNSEKKGVVKYGTEALEKLEFRSRCYGHSL
TPEAVAQVWGPIGGPRIMQIYNEYLHQFVLLRDALLPFANWEEVPFEVEEHAGFKGLRFLEPAREIFLTQLLGKKLTHKS
IVQYAQSVVSSGLIKTGYGFQYRLGTVLPAGLGESARTAARYLLKWHPVQTVAAHEAQDPAAVSFDYEYEDYYAAPRNEA
GAGKPGNKETLPVSGEHYDEPSIARLLPNADTDRSILRFSLDMEGREFAVDLGQLFRGHRFLYRPYGKDNADAAAAKRDS
LSRHLAADILSHSGLVTNTDGIHFIPTGGNELVVWALLGKLYPENVVLLDKGDKQELEAAYVSGKGFGTQFLVL
>lcl|NZ_CP026520.1_prot_WP_241688678.1_3385 [locus_tag=PC41400_RS17160] [protein=CPBP family intramembrane metalloprotease] [protein_id=WP_241688678.1] [location=3757330..3758181] [gbkey=CDS]
MEKNYGLEPPIPFSFKSFAGFIVIFIIWQGGALLWVNLIHALIPAARSGLPSLVTLYLSFIFLILGVYIVTRYFFDIRFR
AFLFEDKRPNYRKFATYFSLYVLALILYLLVDIVVHPQAFTLHFNPLPWLGVLLVTAPLILLQSAAEEILFRGYMYRLFR
SLRGGVFWSIFLTSLTFALVHGLNPEMLNYGIFGPLYYLQGAFFLGIVAYHTNGLAAPIGIHFATNIFNTSIWGYGSSAL
DNMGLQSIVYRHTLDMRFAFFGIFAIVISYSAFQFFKEKRNRL
>lcl|NZ_CP026520.1_prot_WP_042226352.1_3386 [locus_tag=PC41400_RS17165] [protein=TetR/AcrR family transcriptional regulator] [protein_id=WP_042226352.1] [location=3758200..3758808] [gbkey=CDS]
MSRRRLPAEEAKTIILNHAKLLFIEKGYDRATMDDLCVLTQMSKGNLYHHFKNKEVLFLQLLTQHSEQMSQKWLHLAEQS
NSPSQQLLEMADLFGRDCESPLLHALEEYGRTLSSESEALAALREITDAAYRAIKQVLQKGIDQGVFIHEDIETLSFGVM
STMAGATQLCLTMPKLSGEEYAALHVKAIKLLLKGISIDDSD
>lcl|NZ_CP026520.1_prot_WP_042226353.1_3387 [locus_tag=PC41400_RS17170] [protein=MalY/PatB family protein] [protein_id=WP_042226353.1] [location=complement(3758906..3760099)] [gbkey=CDS]
MIKYNFDEIIDRSTTDSIKWSRRHLKENFDDEDNIPMWIADMDFKVAQPIINALISRAEHGIFGYGHKSDEFLDAVVNWQ
QRRNGWNIKKEWILFTPGIIPALNFIVETFCNPGDKVIIQTPVYYPFANIIQNNGCQIANNPLIRNNGRYEINFDEFEKI
ARDSRTKLFFLCSPHNPVGRVWTSEELQRMGEICLKNHVLVIADEIHSDLVYKRNSFVTFGKIAEQLKMKSIICTAPSKT
FNLAGLHVSNIIIPNDRVRAELEHKLASIDIDPSSFASVAQIAAYNEGEEWLEQLLVYLQGNLDFIKKFLEEHLPKVTML
VPEATYLAWLDFGAYNLSDMKLQEMMRKRANVALDDGYIFGTGGERFQRINFACPRSVLQKALERLKKSFEDVEIEV
>lcl|NZ_CP026520.1_prot_PC41400_RS17175_3388 [locus_tag=PC41400_RS17175] [protein=DUF4262 domain-containing protein] [pseudo=true] [partial=3'] [location=complement(<3760122..3760445)] [gbkey=CDS]
MDAQKAVCKKYKADFLAANHDLKLGIALNVKEGIVPIHGLRINPEGDTTGWYIWAGEEFSEDPDFFVPLHVEHIDEWNPE
IKRYLGLAPGWRFIIANNYEDVWFDPNL
>lcl|NZ_CP026520.1_prot_WP_084706498.1_3389 [locus_tag=PC41400_RS17180] [protein=hypothetical protein] [protein_id=WP_084706498.1] [location=complement(3760750..3761079)] [gbkey=CDS]
MTINKKSSRRIKVRDDSYIWTVSPNDGYIIVIVEHEVSRGNRLEIYITSDIDSVWVKFPNTEHLNLKIVRPKDVEFFICQ
ALDLGWIPKVKGKPVVFDFNGLKITRRLS
>lcl|NZ_CP026520.1_prot_WP_042226357.1_3390 [locus_tag=PC41400_RS17185] [protein=nitroreductase family protein] [protein_id=WP_042226357.1] [location=complement(3761168..3762202)] [gbkey=CDS]
MLDFSQTIRERRTIRRFNTTPVAPELLVYLLNGAASLYEAEGTPRWRCIYAGTPESRQRLVESMIAKMKESKLGKLIPVK
MIDFLIKRIKDIPAHLIFIAESAANRRQSDENYAAVCSVMQNFQLLGWERGLGMLWDTDPMLQSESFLAEIGLREGERLA
GILHAGYFDKAPRSRRRTPAKQKWTTIGGGGEPDGPQLDNVRIPAQSILEMLNDAVWAPNDGLREPWRFIYVTGGEAAGK
LRVSYGDASPASLLIVAKEEADPHKREEDYAAVCCLIQNFQLLAKSKPWHVRRTVPEWIYDREWCKPFGIRPQERIVAVL
ELGGDAQSSEPAPAAPQLLNISHL
>lcl|NZ_CP026520.1_prot_WP_241688679.1_3391 [locus_tag=PC41400_RS17190] [protein=hypothetical protein] [protein_id=WP_241688679.1] [location=complement(3762300..3762602)] [gbkey=CDS]
MDYPESIKVLIYNQLLHFFHLQEQRLELELKEIKELVQLHQLKGVPGSLSTIHVLDCIGNHEPINHSAIVDKLNLSKAKR
RSRALSGFWTVTPIVSIRRF
>lcl|NZ_CP026520.1_prot_WP_042226362.1_3392 [locus_tag=PC41400_RS17200] [protein=LacI family DNA-binding transcriptional regulator] [protein_id=WP_042226362.1] [location=complement(3763501..3764451)] [gbkey=CDS]
MANINDIARRAGVSVSTVSRVLNHHPYVSQLKKAAVQRVIDELDYVPNRSAVDLIRKETRSFGVIMPYNNNQAFDQLLHG
ILNRSVERDYTVMVLPTKYNTEKELSYLNKLKNKQLDGIIITSRANSWDAIIPFTKYGAIVACEYTEHPEIGCAYMDRYG
SYLETFQILKDKGHSHVAFTTARGEESNSTQLTIKAYQQIFGNFPPEYHIKDCYSIADGVRAAQQLLQSELNPTAIYANG
DEVATGIYKYARSIGIGVPEDLAIIGQENQPVGIGLELSTVDHQLVKVGELAFDLTIEKSKEKVEIPYRIILRQSI
>lcl|NZ_CP026520.1_prot_WP_042226364.1_3393 [locus_tag=PC41400_RS17205] [protein=FAD-dependent oxidoreductase] [protein_id=WP_042226364.1] [location=complement(3764766..3765809)] [gbkey=CDS]
MVRKQKIVVIGAGIVGASMAYNLAKRNQDVTLIECNSTAACEVTEKSFAWIHSSSRVLQNYRHLYDAALTEYHALEKELT
ELKINWHGALTWGVPASMEESYIQKLNRQQIMELEPNLREYPEEAAYAREEGAIDPIWTTKLLVDKAKEKGAKILFDTKV
MKIRTEDAKVIGVQTSNGFIESDVIVVATGAALPELCKPLGLDVPVAPSPSIIIRMKSNEKLIHTLISNSQFEARQLTDD
TLIAAEDYRDESGENGPEEIGKRTFEILRHGLKGGEKLELESIKVGMRPMPVDGYPIVGFCNQVKGLYLTVMHAAITLAP
VISHLAASEIIDNVRMKELDSCRLSRF
>lcl|NZ_CP026520.1_prot_WP_042227367.1_3394 [locus_tag=PC41400_RS17210] [protein=PepSY domain-containing protein] [protein_id=WP_042227367.1] [location=complement(3766016..3767383)] [gbkey=CDS]
MEKTNYTRFWRWHFYAALFITPLLITLTLSGIGYLFYTNVENNWYKNEFFNQSSQTEQLTIDQGMEKAKQTFKDYSVSKV
IVLEDPYNTRLTMTNKDGDQKYVFLDSNYQIVGSQNPKYTFSNVMREVHSSLFVGGTVVNYLVELAACWAIFLLLSGIYM
TFKGRALKKEPNPSKRQKNLKWHALIGTIITIPMVIIIFTGLPWSALMGDFIYSAAQKYPSIGTPALKQNPPTSDISEIP
WATRKNEPPASTGGSHAEHNGMAAMDHTSNPSGMIAVQQLMNEVKQADISKPYSIIYPKNEQGVYTVSKSSNTGVTGLDV
SPYDEVTAYFDQYSGKLISKVGFEDYGILAKWFTWGIPLHEGHLFGWLNKVLNLVVCLLFLLVIFWGFRTWLTRKKKNSL
SAPPNLSSKLSVGFIIFMLLLGCMMPLFGISLILAVLIEIILSFKKKFQNMNIAK
>lcl|NZ_CP026520.1_prot_WP_042226367.1_3395 [locus_tag=PC41400_RS17215] [protein=M20/M25/M40 family metallo-hydrolase] [protein_id=WP_042226367.1] [location=complement(3767743..3769131)] [gbkey=CDS]
MGIRLANDELGQDEVERPAELPSPVEILQALIRFNTTNPPGEEAACIMYIRRLLENAGIETRIYALDPARPNLLARIKGS
GEAPPLLLYGHVDVVGTDKQAWSRDPFRGDIEGGFVWGRGALDMKGGVAMLVSAFLRAHVRKLPLRGDLILAIVSDEEAG
GEYGASFLVERHADYFTGVKYALGEFGGFSFHALGKTFYPIMVAEKQLCWLKATIRGDGGHGSMGNPGADCMAQLGNMLQ
SLSRNKLPIRAVPAARLMIEGMASALPLLPALLLRGLLRPRLGGVILKLLGEKGETFAPLLRNTVSATVVRGGEKINVHP
SEITVELDGRTLPGVTPQQFMDELRRFAIHDSIGLEVLRHDPCAAAPDMGMFGLLSDVLKEADKEAVPVPMLLPGGTDGR
LFARLGIQTYGFLPMPLPKDMQFTKLIHAADERIPVEAIGFGTDAIYRVLERYGAQPLSSDG
>lcl|NZ_CP026520.1_prot_WP_042226368.1_3396 [gene=fabD] [locus_tag=PC41400_RS17220] [protein=ACP S-malonyltransferase] [protein_id=WP_042226368.1] [location=complement(3769146..3770426)] [gbkey=CDS]
MLNKHAIWFPGQGSQRVGMGKTLSERHPVVKATMEEANEALGMRLDSLMYEGPITELTRTDNAQPAILALGVAMYRVYSR
EWGYLPAMAAGHSLGEITALTCAGAITFPDALRLVRRRGELMQGAAAAGIGSMAAVNGPSPDVVAKVCLEISAGCTDGSR
IVVVSNMNSERQTVISGHKEAVHEASERLSAHGAAVIPLQVSAPFHSPLMAPAAAQFAEVLTRVSFGEMDFPVVSSLTGL
PFGNTAEIPGMLARGLTDPVNWPAVLTFQKSIGVAAAVELGPGNVLKRLAPSDALRVFAFDDKEDEAHLVAASRQTDAYS
IDLLNRCLAIATCVRNRNWNKGEYEQGVVSPYRGVQQLVDRLRETGEQVAEAHVRQALAMLESVFRTKGTSAEEQERRYA
RLQGEFGLRGLPGVSPADRPYAGSLV
>lcl|NZ_CP026520.1_prot_WP_053228619.1_3397 [locus_tag=PC41400_RS17225] [protein=alpha/beta fold hydrolase] [protein_id=WP_053228619.1] [location=complement(3770448..3771116)] [gbkey=CDS]
MYKRWTNRLHPGVRLVPAELAGRGSRMDEPFYADAAEAVRDLLPLVRETALSSDSYAFFGHSMGSLLGYEVLHALREEGF
PLPAAVFLSGRGAPHCEQEKARRIHTLPDDEFLEEVKRLGGMPDELFRHRELLDMFMPILRADFKLVGEYEHRMRPRLPL
RLTILSGKDDDSVKGPLCEWERYATGGCELMLFEGGHFFLHERESDVVSVINKMLTEAPAPI
>lcl|NZ_CP026520.1_prot_WP_042226370.1_3398 [locus_tag=PC41400_RS17230] [protein=hypothetical protein] [protein_id=WP_042226370.1] [location=complement(3771204..3772175)] [gbkey=CDS]
MKLSMERKPFNEFWMNCMLNQAFSIAVSVEPSYRDAAYLNIYRYYPWEAATDKDFRYPTIDTLYYMDDPARFPLSQVIRY
IEPGHFRSKETVPDEIRAMLEGGRNLSVNVDLYDWLPGSMAWKKFHWYHYSLFNGYDKERGTFYVIDDTLAGYEEHEVPE
ERLLKAYGNSEYNVNPSYLGPAFYVYNLHEKIQPYELKLAEVVENAERLARELAEFSIEGMWNVDSDPEKKQAHLTYGLV
GVNIICNRHIANMSLLRSMREKGLIGEALHESLSVQLGAVRDGWDLVKDRFVTGDFERGRELALADDLFAKEKAFWTTLV
AGA
>lcl|NZ_CP026520.1_prot_WP_042226373.1_3399 [locus_tag=PC41400_RS17235] [protein=non-ribosomal peptide synthetase] [protein_id=WP_042226373.1] [location=complement(3772209..3782474)] [gbkey=CDS]
MFQLDLLDTKKNTSGRKGRSGGSVSNVSLKDIAIVGISAKLPRAESHEAFWELLRSGQDLVGGFPVSRKEELKPYLRRME
AQYGSVSFFDGAYIEDISGFDYSFFRLSPKEASLMSPAQRLFLQTAWGALENAGYGGDALHGSRTGVFIGYNGDALHDYK
RIIEALDPDALSMAAPGNLSSMIPSRISYLLNLRGPAISVDTACSSSLVAVHLACQSIRSGECEAAIAGSVKINVLPLDT
GIRLGIESSDNRARTFDDSADGTGIGDGVIALLLKPLSRAQEDGDPIYAVIKGSAMNQDGSSVGITAPNALAQEDVIARA
WQDADIDPETVTYMEAHGTGTSLGDPIEIDGLTRAFRRYTDKRQFCAIGSLKTNIGHLDHAAGIAGLLKAILSLVHKQLP
PTLHYRSPNRNIQFVDSPVYVNDELTPWETDGEARRCGVSAFGMSGTNCHVVLEEAPAGIPSARSDRAELFCLSAHSMTA
LERLVLLMRDWALQHRDDESRSLADLCYTLAVGRGAHRYRLAIAVSGWEELAETLHAAAESGLTGLSLPNVRFGTADSAA
AGNMRQQPVPNERVMEALELGQRYVGGYEIDWKAIYLGQRRKRLPLPTYPFDSHRCWVEEARAEAALPTPFAAWRGNSLG
EDIPEELRQEINETFARWQTALERSGGMTSKRRSGVTLTGERSGLEWEQQIADVWGELLGYDELPADAGFYELGGDSIIA
LKIVNRLSELAGVRIQATDLLGHADLPSFTALIERRLRESGADTSRSTNAQHDGRGTDVGNGGHETNSGHDGQGANGGRD
RAGADGGHVGPLTGQASDSLSGDTLEPIPKAAPQEHYPVSSPQKRMYLQQQAMPDDRSYNLPELLHLDGAIDANRLEAVL
QAIVERHETLRTTFHVIDGEIRQLIHAATPFRLIRMEADEAEADGVLQNLYQPFTLDTAPLLRAALLTLGPERHVLFLDM
HHIASDGMSAGILLSDLMTLYQGASLPPLSLQYKDYAVWQESRPLSGNAERYWREQCRGDWPTLALPTDAPRPPVKSNRG
ETFDVYVEAGLTAAVRSLAAESGATPFMVLLSAYYTFLAHYTGAEDIAVGTPIAGRNRRELEAIAGMFTGTLALRAYPRA
DKPFRTFLSEVKEMATGAYAHADYPFEEIAQWVDRRDASRSPLFDTMFIMQNLGIPNASSAGLSYRHQRFEHGTAKYDLM
IQAVENGAELRLVVEYNADLFRRATAERFMRHYVQLLRSLTLRPDAALGEAEMLTEEERLHMLALGRREADFPPPRCLYD
CFKEQAAAVPGLPAVSCGGESLTYRELDRRTDALARTLRAKGVGRGSIVGIMAERSIPMLTAMLAVMKAGGAYMPVDPHY
PEERILYMLEDSGADLVFVDGSRSGAAGIVPASLQVLDIADETLFSGEDDAVAAWPQDAADPMYVIYTSGSTGQPKGVIV
PHRSFYNFGHSLRTFFDGQYGERDRCLSLTNISFDVSVAELFMPLMFGACTVLYPEPKLLDPRRIAEVIAEERITFAYLP
PSLLKEVARLLEQSASAITLDKMLVGVEPIKDETLELYTKLNPGIRIINGYGPTEATVCSNMYAYKPGAYRGGYVPIGGP
MHNVEIYIFGYGGQLAPIGAAGELYIAGSGLADGYVNKPDMTAERFLAHPYREGRFMYRTGDIAKWLPDGNAMYVDRADQ
QVKIRGVRIEMNEVRSQLISLPEVEDAVVVVREQASGDKELCAYVVASDKLPSREWRRKLKDKLPEAMIPAYIAAIDAIP
LTPNGKVDRRALPEPEFGLRERGKPELPRDGMEERVAAIWEEVLGDTAGPIGIHDDFFELGGHSLKAAVLAGKLQQATGA
LVPLSQIFELTSIAEMAAWLKERDGAAAAAEPIPKAERREWYPMSRAQRRQYMMQMLAGDATMYHVPFALRLQGRLDAAR
LEDAFTALIARHESLRTTFHMTKDEFRQTVHPPYALKLETLDYTALTGDEALRGALPDGAEVTRLMKVFQRPFHLDKLPL
LRAGLLRLNEEDHLLLLDVHHIITDGVSSGLLVNELSQLYAGQALPELRVQYSDYAVWQEAGFENAAYAEDELYWMEIFE
GKLPVLELPTDRPRPSLPSYKGRRHSFRLDARTTAEARKLAREAKTTLYTVLLGAFAAMLAKYSGQAEVIVGTPAAGREH
ADTHGMIGMFVNTLALRTRPERDKTVRAYMAELHNHVVEALSRQTYPFEELVEGLKADSDRSRNPLFDVMFVLQNMDRAV
MEADGVVFGEMPFEAGTSKFDLTLEAVEREAEIELQLEYATDLFHEETARRMAESYTVLVREMCKSQGRTVGELELMSES
EREQLLALLSGGGTLLQAASREGSNGRMEIRAEKGMTDRQGDAAAAFTQATANWTGFVQEVERQAARTPDATAIQFGAET
VTYAELNERANRLAHSLRAKGVGSERIVALMASRSPLLLVGILGILKAGGAYVAIDPAYPKERIDWMLEDCGEALLLTEK
AYAGVVTRAVAEWYLDDPELYAADRGNPDFVHTPDHLAYVLYTSGSTGRPKGVMIEHRGLSHFISGFRNRIPFEKGQSIL
AMASASFDIFVVENLLPLTLGMRIVLASEEERGDAALLQGLIDAHDVDVLQITPSRFKWWMNQRGETDGLQKLRILMIGA
EPLTADVLGRLRAATGARLFNLYGPTETTVWTSIREVTEGEDISIGTPIEGSVMMVLDEGLKLQPTGVTGEICIGGPGVG
RGYLSHPEWDVAFVPNPYAPRERMYRTGDLGRRLENGEFAYAGRRDFQVKIRGHRIEIGEIEQLLLRHDQVKETVVTAFQ
EEDEEYALCAYVVMQNAADADAEPDGAAPSDPLTRQLREYLVDKLPSYMVPAFMVVLDAIPLTPTGKIDRRSLPKPGPAD
AGKRELVPLSTDTERKLAEIWKELLGTDAISAKDSFFDLGGHSLKAAGMISRIAERFGIQMPLRDIFMNPTLEGLAAHLD
NSGTMAAARIPKQPEREHYPMSRAQRRQFMMALISDEATMYHVPFAVHMQGKLDVERLEEAFLAIMRRHESLRTTFHHEE
DEFRQRVHAEPVFTLEQGARLRMAAGKLLAEGGDEGLFGGISALMERFIRPFDLGSLPLFRAGLIPLGEERHLLLLDLHH
IITDGVSVSVLLQELTTLYGGGELPELDIQYRDYTVWQEEQIGSEAYMGHERYWLEAFEGELPTLELKTALPRPELQNFK
GSLINFRLDEELTLKIKAFARERGTTLYTVLLGAYSMLLSKWSGEEDIIVGTPVAGRNHAQLEGLIGMFVNTVAIRSFPE
SYRTVGDYLSELHEDVLRALEHQNYPFEDLVQKLGIEQDRSRNPLFDTMFILQNIDRVAYRSGGIEFDPKEFDPGVSKFD
LTLEAAERDGRIGFTLEFATSLFREENVWALADDYTAIIHALVEDGAKKKISAVLAASKLA
>lcl|NZ_CP026520.1_prot_WP_241688680.1_3400 [locus_tag=PC41400_RS17240] [protein=CPBP family intramembrane metalloprotease] [protein_id=WP_241688680.1] [location=complement(3782488..3783258)] [gbkey=CDS]
MYLVIYAAIVMLVNQALYNWVGDPGLKEWIAGNSGIVLIVSNIIVLAVYLPLLRWQKISLRDLGFVPARGGSLLLSAGTG
VWLGLFIATFTRLPWIKATFPAISDLVAFVAGGSLIIFVLGSLLLGSLLEEWLFRGMLFHTLRQRLSVPWTVLLQAVLFG
AVFMNVTVGAFAALGAIVYGAVRAGSRSLWGSLAAHVFSTGTLYIVLQWAGDWQSGTVGLLALISGLGIAAHVLLLLRGS
GGRVEKGRAVSDSITS
>lcl|NZ_CP026520.1_prot_WP_042226377.1_3401 [locus_tag=PC41400_RS17245] [protein=type II CAAX endopeptidase family protein] [protein_id=WP_042226377.1] [location=complement(3783288..3784097)] [gbkey=CDS]
MKAWGTMLARVAVVIGLYFAWFFTVTTLFFDYVYPRSTWFAQNTVTVIILNDMVGLPLMLLAWRFIFKENLFKAAQFRVM
GGKSVAIALWIGLGAGLFTVAFSRLPVIASDKYRFNELFDYLNRAEWYVFLVFLILGNIYKETLFRGILMNEFRRVLPVW
IAIVIQGVLYGALFFLGDIPLSLYGFLGAVIFALLYVWFKSIWAPITAQVACQGSQYLLWHYGPKTTDITVMSIMMAVAA
TMIGVGIFLAVKHRSSVSAPLSSKAVTSA
>lcl|NZ_CP026520.1_prot_WP_042226380.1_3402 [locus_tag=PC41400_RS17250] [protein=CPBP family intramembrane metalloprotease] [protein_id=WP_042226380.1] [location=complement(3784102..3784974)] [gbkey=CDS]
MKKVLAMIGNIALYLGVFYALLYLLRSTYNYEVTLSFAKFLDRNPAMFMVVLFTLITLVYMLIFRIKGWIWPHAEKNLFR
ASGFKRLSASRVLLMIGLGLAGSLFSIGLIVIDDIARQFPSIPGLVDDMIRGDSIWYVILGAGLIGPAFEEILFRGLIFR
ELRRVMPVYAALLLQAAAYAYFQPSAALSVISIGSGLIYGSLYLRTGSLWAPILVQNTAMGTIFLFKYIGFYEWFDKLGD
VSLYIITALCLAALIGGSVYVWRTSGNGGIGARAGQALTADAPSAAQKGV
>lcl|NZ_CP026520.1_prot_WP_042226382.1_3403 [locus_tag=PC41400_RS17255] [protein=non-ribosomal peptide synthetase] [protein_id=WP_042226382.1] [location=complement(3785281..3797433)] [gbkey=CDS]
MRTIRNLSGVLLARKENDAGILFCESGGERYLSYGKLHASAGRALHLLREQGLGPGDFAVMQTEDNERFILLFWACVLGG
IVPVPVTAARTDEGRKKLIQVFKQLEAPTLLCDEELWPGIQGYALKHGEAEALAQLESGSVPIAEVLAAAADAGRPEAEP
YDAAETDTAFIQFTSGSTGDPKGVVLTHGNLLSNMRAIIHGAQSTERDSSLSWLPLTHDMGLIGFHLTPLFCGMNQLHLP
TSTFVLHPMKWLELTHKHRVTCLSSPNFGYVHFLQHWKPEGAAAWDLSCVRLIFNGAEPISAAHCRDFLEAMKPYGLKEN
CIFPVYGLAEASLAVTFPGTDDHLNTVKLDRQSLAVGQPVQMAAHGDQTAIEIVELGFPVQECEVRICDEAGNPSAAGST
GLIHIRGRNVTSGYYKRPDVNKQTISPDGWLNTGDLGFLLEGRLYVTGRMKDVIFVNGQNVYPHDLEALISGLKGAEIGK
TAIAAVQDATTKRDAIAVFIQHRGKTGGFVPLMSEVKKLLNGSAGLDVAFVVPVRRIPKTTSGKIQRYVLAEALREGEFA
EALAELTELETAAALKRCDAESGDALCGDVGLEQATKTEERIISIWREVLKKDYIGRKDGFLEQGGNSLKAAYAAARLQE
AFGVELTLTELFLHDTVSSLAALLEQKAQEGEGTAAVVAPPVAKAEERTVYPATSAQKRLFILEELNPGLLNYHLPFTVQ
VKGPLDIERFRSAWRALIERHPALRTSFYREEDNVMQRIEQGVEPPLAVVDGSGWTAAELTLRQRRFLQPFRLEQAPLLR
MELVRLEHESSVLFLDMHHIITDGTSMGILMSDLTRFYEGNAPEPLSIHGGDMALWEEAGRSGGWLERQKLYWREIFRDG
VPSPSLPSPRPRPQTSSYRGAIERFELEPELAGSLRALARREGVTLYTVLLSIYYALLAKYTGESDIVVGTAAARRVRPE
LQSVVGMFVNMVPIRLRGQSGLRFDQWLKQVNGEALAAIGNGDVPYEEIAELANVKREYGRNPLFDSVFTLQNMELPEWR
SKDVVYTPQTMDSSSAKFDLTWECVNNGAGIAFTLEYALDLYDERAAAQLARHYAALARAIAAQPHTALGELDPVDAEER
ERLLRLGRSSAAPPQELSLHRLFERQVMETPQQIAVEMGELAYTYSELNERANRFARALARRGVAPGAKVALLLLRSPDM
IAAILAVLKAGAVYVPIDPQYPEERILFMLQDTETQAIVTAAATMEMGARLAGDSTDVIDMDELDTGWASEYGHDLDIDG
SADDLAYIMYTSGSTGKPKGIMTSHRNVSRIALQTDYVSISENDALLQLSNYAFDGSTFDLYGALLNGAKLTLVGEEEVA
DVMRLTELIEGRGITLFFVTTALFNTLVDHGLESLRGLRHILFGGERASVPHIRKALRALGPGVLLHVYGPTETTVFATC
YPITEIGDGQGTATVPIGTPIARSETLVLNEQGRLQPDWVPGELWISGDGVASGYVNLPEQTASKFRPHPFREGETVYGT
GDLVRRLPDGNLEFLDRIDSQVKLRGYRIELGEIESRLMDCAGVKEAFVALIENDGAPYLCAYIVAEQTDAQTSVRSQLT
GKLPAFMIPSALVQLEKLPLNASGKVDKKRLPAPERAVSYEAPSTETEIIVAELWQEVLHTGQVGALDHFFEHGGHSLKA
AALTSAIYKHCRVRMPLQEIFKLPTVREQAAWIDRAAKEAYEPVTDAPKAASYPLSPPQRRMFLAESMADIGLTYHIPLA
LKLSGAVGETRIGPALQKLIERHEPLRTSYHWIDGVPAQIVHDSAAFALQTAELAETTLPECLSEFYRPLELAAAPLLAA
CYCRPESEGAEAYLLLNIHHIAADGISVKLLLEELTALIDGAELAPLALHYKDYAVWQERQSGSERARASRAFWTEQLAE
PAAPLDMPLDRPRGDRQTFAGDTYTFHIPAALAARLKETAASAGASMNSLLFAAYALLLKGTTLQSEFAIGSLAAGRTHP
DTARMIGMFNQFLPIRMRVADELPILAFVRETHRRLLAAYEHGDVSFEHMMEAARYPHDPSRNPLFDTMLIVHNQFEDEK
LAARGDGWSMEPLPVSHGTSKLDFKLDLYSDGEGGLRAELEYNTGLFYKVTMERLADRLLYILEQTAASPDLICGDVVMA
TPAEEAAFAAWNDTGHSYPAEKTIHGWFAESARRWPDYTAVRSAEGSLTYGELNDRAERMAAVLRDKGVAVETIVPIAAQ
RSLSMMIAIMAVLKAGGAYLPIDPEHPPERIRGILEDSGAKLLLAGAKWIHTLPAFQGETLDLDALAATAASELPKGEEE
TDLSPQAGPEHLAYVIYTSGSTGKPKGVMVEHRAAINRLNWMQTAYPLNERDVILQKTPITFDVSVWELFWWSFAGASLY
LLEPGGEKEPSVMLRAIKEQAVTVMHFVPSMLGVFLPYAADSGRAGELCSLRYVFASGEALKPAHVAGFYSLMKRGCGQS
RLINLYGPTEATVDVSYYDCLDAGAGTIPIGKPIHNIRLHIVDERMRPQPIGVSGELCIAGAGLARGYRNRPDLTEQSFV
PNPFAAGEKLYRTGDLARWLPDGNIEYLGRFDHQVKLRGLRIECGEIEHALLLQTEVRDAVVTAVTDHAGEPALCAYLVT
ADGEMPEEASLRAALKELLPEYMIPSYFVAMEAFPLSPSGKADRSRLPQPRFAEPSACGEAPATPTEQRLAELWREVLGL
ASDEPNDPATIGREAHFFQLGGHSLRAAQLAGLIEQSYGAAFAMKDVFDAPLLRDMAARIDHAVPKRAAVIAPVESRPFY
PLSLAQNRLFVLQQLYPDSTAYNLPLALNLTGKLDPDRLRQAIQGLIDRHEPLRTSFDWADGKPIQRVHERVTFELAVHA
LMDAADLNTFARSVVRPFDLRKAPLIRAFLATDGESRHTLLLDIHHIVADGVSMNVMARDFEQLYRGGSLQPLPIHYKDA
AVWQQEWMASEAREQQEQYWAFRLAGTLPVLQLPTDYPRPPEQSFEGAKLALPIPAELAEGIGAAAEQWGVTPFHLWLAA
YHTLLHLMTGQDDLIVGTPIAGRFHPATEPLVGMFVNTLPIRSRPSGELPFRAFAEQLKETALSALGNGLLPFEHMVTML
DVPRDLSRNPLFDTMFVMQHMEAGQAAADALQFEAQVPDNRVSKLDLTFEIADQGTKGAALTIEYATALFKEDSIHRMAE
WYFRIVGLALAEPERPLGGIGLQGAAAAEKQSEAFNRTAAPYASGDTVEVYLERQAAETPQAVAVAAEDGALTYAELNRR
SNRLAQALRKQGVKREERVAVVMERSLEMMIAIFGVLKAGGAYVPVAPGLPPERIRYMLENSGAKLVLTKGKAPEGLETF
LPFMDVHEALAQERDDLPVEKVHDSRSLAYVLYTSGSTGQPKGVMIEHHSVVNRIGWMQKQYGLDASGVILQKTPITFDV
SVWELFWWSFAGAKLVLLPPGGEKDPSLMMDTIARHGVTTMHFVPSMLHLFLEHPGLLRGEARLESLSHVFASGEALTAD
QVRRFRERIGVPFDARLVNLYGPTEATVDVSYYDCSDGDIPIQVPIGKPIDNISLYIVSEAMKLQPVGIAGELCISGAGL
ARGYINRPDLTEEKFVDNPFAAGSLMYRTGDLAKWLPDGTIEYLGRIDHQVKIRGQRIECGEIEHVLLGHPSVSEAVVMK
REAASGSEYLCAYIVCSEPADHQELREFAALRLPDYMVPQAVVELPAMPLSPNGKIDRKALPEPELAADSGTPFVEAASD
IELAIAAVWREILQRDDFGIHHRFFDVGGESLLLVRAHQRLEELYPGALGVTDLFTYPTIASLAAYLESRNRELMSWSWS
GLPVADDCVSSGYTAERTGSVQFQLDRTVAEGLAELAASRSVTLEAAAFAVYLYFWRGESGMGRLVLPAMTDNGLIAPKE
IDFAEVRDFDTLLRHAAKRASAGECYTARQIRRQPQAEAGFALFPLFAGAAQHGFKEREALLERFDVVLYMDGAAAPSGA
EGPELGGVWTYNARRLGKEAVLDWASAYQELLLSVVEQYRAATGSTVSGA
>lcl|NZ_CP026520.1_prot_WP_042226385.1_3404 [locus_tag=PC41400_RS17260] [protein=aspartate 1-decarboxylase] [protein_id=WP_042226385.1] [location=complement(3797462..3797845)] [gbkey=CDS]
MFREMHKSKIHRAVVTEANLNYVGSITIDQDILDASNILENEKVQVVNINNGARLETYVITGPRGSGVVCLNGAAARLVQ
PGDRVIIISYAMMEESVARTYRPHVVIMDDHNRIVTEDYQELHSTTV
>lcl|NZ_CP026520.1_prot_WP_042226387.1_3405 [locus_tag=PC41400_RS17265] [protein=glycosyltransferase family 2 protein] [protein_id=WP_042226387.1] [location=complement(3798508..3799569)] [gbkey=CDS]
MEPLISIVMPTYNRAQIISGAIESILRQSYANWELIVVDDRSTDHTEETIREWTRKDVRIRYVCNEREKGPGGARNTGML
AAQGEYLAFLDSDDEWYPRHLTDSMRTHAQTKADISFALWLERHGDITSYNFDNKVERDLLHSMRTTFETWNDGETIVFE
KGLFEAFLSHTRNFFQLNTMVFRRELLDEVGLINEHFYLGEDTTYLVRFFDKYRIALQTKPHSVYRESPDSLYFFCDRWQ
LDPDTIHLNEDLYKKIEGLSFKSIKVREHIRDLVAQSESFSNKAKQLSYIDIGIASKYYTLSYLNRYDRKKALRYCRRSL
RYKVSIFNLFLFVKLLFSSKSGNVFLQRALNIW
>lcl|NZ_CP026520.1_prot_WP_042226389.1_3406 [locus_tag=PC41400_RS17270] [protein=MarR family transcriptional regulator] [protein_id=WP_042226389.1] [location=complement(3799832..3800260)] [gbkey=CDS]
MQSRKDTPYLDLFQIIGLKLKKRADESIKELGLNAQQGKVIDYIYENQDNHLIQKDLADRFHLRGASITSMLQGLEQKGF
IERKIPANNERQKNIYVLPKAIELIEDFQDSFQKVEEEIVQALTDEEKQVLKKLLIKINERI
>lcl|NZ_CP026520.1_prot_WP_042226391.1_3407 [locus_tag=PC41400_RS17275] [protein=amidohydrolase family protein] [protein_id=WP_042226391.1] [location=3800424..3801734] [gbkey=CDS]
MENVAQNAALEQVTAITNVRIFDGEQIIAPRHIVIKGKSIISVGGDLPADAILIDGENATLLPGLIDAHVHTSIGGLRDA
LKFGVTTELEMNGDFTKRGREIQLKNVNDVADVRSAGTAITAPGGHPDELLPDGDEIPEFVLKELEKLSEEDREAMLAAY
AHDHEEIPQVTTVEEAIKHVHTQVENGADYIKIMIEEGTVMGAPGLPVLSDEILKTAVTEAHKFDKLVIAHVLTARSSQE
AIDFGVDGLGHLFIDRPEYTSELVKSIADSGAFVTPCLVLNSSIIGNPASELANDPRVHSKLSPDWIDILNSSFNTYPQG
NMENSFKNVMDLHRAGVDILVGTDVAPVPVPNLGGLAHGASVHHEMQLLVKAGFTPIEALQSATSKPARCFGLHDRGRIT
EGARADLILVNGDPTTTISDTLSIKSLWFNGTQQLG
>lcl|NZ_CP026520.1_prot_WP_241688681.1_3408 [locus_tag=PC41400_RS17280] [protein=LysR substrate-binding domain-containing protein] [protein_id=WP_241688681.1] [location=complement(3801846..3802055)] [gbkey=CDS]
MQTGLAATLLPASVLTGECADLGIYPIPEKYRFTQTSLIHRKDRFKSKAFTAFSDMIRESMAKADDLVG
>lcl|NZ_CP026520.1_prot_WP_129112506.1_3409 [locus_tag=PC41400_RS17285] [protein=hypothetical protein] [protein_id=WP_129112506.1] [location=3802312..3802656] [gbkey=CDS]
MLKKKWLVLPVLSLSMMTVFGFSSAVLASTSTQTQKSSVVSESEKASFNKIYIDKTQRLYPKTSYPTFNDTYDTYHYYDG
QYEGELNRDSAYTEDKYFFYVTYSGWAYNTDLQS
>lcl|NZ_CP026520.1_prot_WP_042226394.1_3410 [locus_tag=PC41400_RS17290] [protein=hypothetical protein] [protein_id=WP_042226394.1] [location=3802966..3803340] [gbkey=CDS]
MKRKITSIAFMLVLLLSFASTSFAASSGPGTGNNTWAPVQAHGNIYIEGNFINDGQISLGVNFRLYRVTPNGDVLVGSNW
TNTIVAGQTARLKIPLASNQPDGQYRLVLDSPGDFTVIAYLGYN
>lcl|NZ_CP026520.1_prot_WP_042226396.1_3411 [locus_tag=PC41400_RS17295] [protein=fructose-bisphosphatase class III] [protein_id=WP_042226396.1] [location=complement(3803521..3805452)] [gbkey=CDS]
MDAQFLDLLAQKYDTEEKVVTEIINLEAISNLPKGTEHFVSDLHGEYQAFQHVLRNGSGNVKEKIKDLFREVWTDNEIND
FAALVYYPEEKIQLVKAELCNQQALNQWHRQTIEQMLMLISYASSKYTRSKLRKALPEQFVYIVEELLYKTDSTNNKDPY
YDEIYRQIISLGQADKLIIGLAYTTQRLVVDHLHVVGDIYDRGPDPDKIMDTLINYHSVDIQWGNHDVLWIGAYAGSLVC
LANIIRICARYNNLDIIEDVYGINLRPLLNLAEKYYDNNPGFRPKLPADHNVSESEILQITKIHQAIAMIQFKLEIPIIK
RRPYFHMSERLLLEQIDYDKNEIKIGGKTYPLENTCFATVNPQHPEQLLEEEQQVMEKLLFFLQHSEKLARHMNFLMKKG
SLYLKYNGNLLIHGCIPLDEEGNMEKMQIEDKTYSGRQLIDVFEYYLRYAFAHPEETDDLATDMVWYIWTGECSSLFGKR
EMTTFERYFIQDKETHKERKNPYYHLRENDEICRKILLEFDLNPDHGHIINGHTPVKEIRGENPVKANGKMIVIDGGFSK
AYQSTTGIAGYTLLYNSFGMQLVAHKKFNSKEDVLCNGTDVLSVKRLVDKELERKLVRETNVGEKLLQKISNLKNLLEYR
YMK
>lcl|NZ_CP026520.1_prot_WP_042227370.1_3412 [locus_tag=PC41400_RS17300] [protein=DJ-1/PfpI family protein] [protein_id=WP_042227370.1] [location=complement(3805724..3806323)] [gbkey=CDS]
MEQVRQTRNVAVLIFEQVEPLDFAGPFEVFIAGSNSGQDFNVFTVAENDSPIKASGGLSINPAYTIANCPKPDILVVPGG
WGSRKEMNNETITNWIRETSNDVEIILSVCTGALLLAKATLLDGLKVTTNRLAINELRDVVPTSTEILEEARYVDNGKIV
LSAGVSAGMDAALYVVEKLLGEDTAIEAANIMEYDWKRD
>lcl|NZ_CP026520.1_prot_WP_042226398.1_3413 [locus_tag=PC41400_RS17305] [protein=erythromycin esterase family protein] [protein_id=WP_042226398.1] [location=3806757..3808061] [gbkey=CDS]
MNSNDQSTGQNAYLKELNALFIPFNEMENLDRIVEAIGDADMVLLGEASHGTSDFYRVRAELSKRLITSKGFNIIGVEGD
WPAAYRLNRYVKGKAGADSSAAEALGDFTRWPTWMWANTEIRDFAEWLKAHNAASHGNGSERQKVGFYGIDMYSLWESME
AVIGYLEKTRSPQLEQARMTYACFEPYSRDHQTYGIAAGLLSESCEDEVVELLKQMRSRRMAHPGSEEELSAEVNMLVTS
SSEAYYRTMVRGGPQSWNVRDRHMVESLKRIMSYYGSSAKAIVWEHNTHIGDARATDMAQEGMVNVGQLLREDPEQRVFA
VGFGTYQGEVLAGSAWGDPVEKMTVPPGQPGSWEDYMHRAGGGVDGMVILSEASSVFQTVIGHRAIGVVYDPNHEQFGNY
VPSSMKDRYDAFVHLDRTTALHPLPILRPVAEKV
>lcl|NZ_CP026520.1_prot_WP_042226401.1_3414 [locus_tag=PC41400_RS17310] [protein=MFS transporter] [protein_id=WP_042226401.1] [location=complement(3808288..3809493)] [gbkey=CDS]
MSRHSSSTGTAQAAVMPARLPWMALLALAMTGFICILTETIPAGLLPLIGDGLHITTAMAGQLVTLYAIGSLVAAIPFAM
LTGGVRRRPLLLAIIVGFLVFNTITALSTSYALTLIARFFAGVAAGAAWGMIAGYARRMVPDALKGRALAVAMVGTPIAL
TFGVPVGTLLGDQIGWRAVFVLMSFLALILIGWVLWKVPDYPGQPANQRMTVRQVFVMPCMRPILTVVLTWMLAHNILYT
YIAPFLERSGLAGRVDLVLLVFGSAALVSIWIVGILIDRILRMLVLISLSGFALVSIALWIGGTSVTIVYSAVALWGFTF
GGAATLLQSAQANASGEEGVDIAMPINTTVWNLAIACGGIVGGALLELLGVQSFPGALLILLIVSLMVAWSASKHGFPGK
Q
>lcl|NZ_CP026520.1_prot_WP_042226403.1_3415 [locus_tag=PC41400_RS17315] [protein=TetR/AcrR family transcriptional regulator] [protein_id=WP_042226403.1] [location=3809620..3810249] [gbkey=CDS]
MPRTGRPRAFNRDEAVAAAMFLFWEHGFESTSLTQLREAMGGISAASFYAAFESKEALFYEAMNVYKSTYGQVSDSFRDR
SLTPKAAIEQVLRRAARMQTDRSHPLGCLLVLSAVNCSPELNHIQEMVAKERQKVRGWLKDCLSRAAEDEELPPTADISV
LVALFDTFLQGISTQARDGIPYETIDEAITQLMRIWDVSSAKPDIQGIH
>lcl|NZ_CP026520.1_prot_WP_241688682.1_3416 [locus_tag=PC41400_RS30090] [protein=hypothetical protein] [protein_id=WP_241688682.1] [location=complement(3810479..3810688)] [gbkey=CDS]
MFDGAELGKATPAHPDNVEAALSTYEEALFPRSEAATTVAHQNHELFCFDDRVPFGLIDILVSKERFSS
>lcl|NZ_CP026520.1_prot_WP_241688683.1_3417 [locus_tag=PC41400_RS30095] [protein=hypothetical protein] [protein_id=WP_241688683.1] [location=complement(3810724..3810984)] [gbkey=CDS]
MTTPVTIVGDGLGGLTLARVLHVHGVPATIYEAEPSAEARAQVGQLDIHEYNGQLALQAAGLFNEFLGLIHKGGQAVRVL
DTHGNT
>lcl|NZ_CP026520.1_prot_WP_042226406.1_3418 [locus_tag=PC41400_RS17325] [protein=MDR family MFS transporter] [protein_id=WP_042226406.1] [location=complement(3811027..3812445)] [gbkey=CDS]
MSEKSSLTFNRTLLACIFLIGTFVTILNQTVLFTALPEIMIDFGISAGTGQWLTTAYLMANGILLPVTAFFISKYTTRQL
MIVAMSLFIAGTTLAAVSPNFTLLLIARIIQACGASITMLLMQTVFFIIFPREKRGAAMGLVGLVVAFAPAIGPTLSGLI
MNHFSWRYIFIMVLPVAVIVLILTLLYMKNVTEVNKRQKIDIPSVFLSTVGFGALLYGISIAGSGSALSAALWSIVGAIG
LTLLILKGLRSEKPMLEFRVFKSYAFVLSTIAVVISFICLMGPQTMLPIYIQNIRGMTALESGLILLPGAVVNGIVSMIA
GKIYDRIGGKYLGVIGYILIAVSVIPFIFMHAQTALWMIVLFYALMIVGISMIMMPMTTEGLNALPDHLINHGTAMINTF
RQVGGSIGIALLITIYTEVSSHAVESGRLNEVAQSQVYGINYAFIGIFVFGIVGLVGSSLLKSRNGKAAVGN
>lcl|NZ_CP026520.1_prot_WP_042226409.1_3419 [locus_tag=PC41400_RS17330] [protein=MarR family transcriptional regulator] [protein_id=WP_042226409.1] [location=3812685..3813152] [gbkey=CDS]
MDLSKENAVLAVEVLDLLFRATRHIQQEYEMKLESLSLPFQLSGPRLRLLLVVWRAEKIRMNELASKLGVKPRTVTELVD
ALERDGVLHRTPDPNDRRATLLHLTEEALNQISRVRSVQMQISDEILQKFSAEQRTQLHALLSKIFEGMEFDFVY
>lcl|NZ_CP026520.1_prot_WP_042226410.1_3420 [locus_tag=PC41400_RS17335] [protein=glycoside hydrolase family 64 protein] [protein_id=WP_042226410.1] [location=3813508..3814854] [gbkey=CDS]
MRKGMSYLLILSLFLTCFSIFATKAEAADYTRGVELSGSAATIWFKSAVNTSWVDVHYKVNGGEQQNFRMAYNAGKSRYE
KAVSNISAGSVITYSFTYNNGTPAYDTEVFTYTAGTTPPPSSSGSIYDILPSSIPSPPSGAVSVKVMNGTKGAYSDQQIY
WGVIGINPANNKWSYLDLNGNLIPISTALNDAPGHLTKKGISYANIYNKVSDASWVTLPKITAGRMFLSVGTPLYIKTYN
DGFAGPDINNPTDPNRDIYFDFAEFTIDAAGYHGNTTRVDGFGFPIQHRLVNKSGSFDRTTGELESETRSGIFTKFQAEV
PAAFKSLAIVQAPYRIVAPIHGSFAQGGANANYFGGYAPYSTQDIFRCDGALTNAATCAAINRHVYTTSDWNNVSNYYKA
APANYYAKFWHDHSINRLAYGFPYDDVNGQSSYLEVGDPKGLIIRVGW
>lcl|NZ_CP026520.1_prot_WP_042226412.1_3421 [locus_tag=PC41400_RS17340] [protein=MarR family transcriptional regulator] [protein_id=WP_042226412.1] [location=3815160..3815633] [gbkey=CDS]
MNKEEQVLMGFRDLFNKLVWLNKGKMENSLKGYKPSEVHCIEYIEKNVDSNVTKLAESFYMTRGAISKITKQLLKKGLIE
SYRKPDNKKEIYFRLTEKGKVIYNIHEDLHKEFQERDKAVFEQITEEQFDSMLSFVETYSRHLDAEIKKRGISIEPE
>lcl|NZ_CP026520.1_prot_PC41400_RS17345_3422 [locus_tag=PC41400_RS17345] [protein=MFS transporter] [pseudo=true] [partial=3'] [location=3815789..>3816412] [gbkey=CDS]
MFKFKSPHEQDTEQTIDKHALLFGLISVFLCGIGFSIITPVVPFLVQPYISNPGEQAIVVTLLTSVYALCVFFAAPVLGA
LSDKYGRRPLLLVCLLGSAIGYFVFGIGGALWVLFAGRIIEGITGGSIGTIFAYFADIIPPEQRTKYFGWLSAIVGVGTV
IGPTVGGLLAKFGYSVPMYFGAVITLLNVVYGFFFMPESLKAKNRLKE
>lcl|NZ_CP026520.1_prot_WP_042226414.1_3423 [locus_tag=PC41400_RS17350] [protein=TetR/AcrR family transcriptional regulator] [protein_id=WP_042226414.1] [location=complement(3816639..3817319)] [gbkey=CDS]
MKKQQPQISEDKILETSWELLGEGGIEKFSMRRLADRLGIQAPSLYWYFKSKQDLYQRLANQVSKMILEEFRSEGDWKEQ
LTGLAVTVRSVLSRYPCSTQLMMMTLPHEPDIIRFTNRMLLCVESTPLDQEQKMQVILTLVNYVYYFVLDDYQHQRNVTA
ILKEQKELPVGEMDRLLDSMSETDAGLFRRMFTNGLFELMGTDKAFEFGLRLILLGIEQVIKEQGK
>lcl|NZ_CP026520.1_prot_WP_042226415.1_3424 [locus_tag=PC41400_RS17355] [protein=MATE family efflux transporter] [protein_id=WP_042226415.1] [location=3817461..3818813] [gbkey=CDS]
MDAENLHYFEKAPIAKAVAHFAVPMMLGTSMSVIYSILNAYFLGTLGNTAMLTALALTLPLFAAIMALGNLIGMGSGTFI
SRLLGEKKYEVVKHVSSFAFYSSLVLGLIVMAVGLPLIDSIVHGLGATPDSFGFTKDYVTIMLIGSPFVVLFFTLENIVR
SEGAAITSMIGMILSVVVNIILDALVIFVFHWDVIGVASATVVSNLAASIFYAFHMRAKSQFLTVSIKWFKVTKDILSNV
IKIGVPVFIMSIFLGAMSLIFNHFLVEYGDQSVAAYGISSRLLQFPEFILMGLCEGVVPLIAFSFTANKLRMKHTIGFTI
KVIVALAVLFGIIVYLISGHLIGLFTNDPQLIEMGSYILHVTFLSLFITGMTTLFTGIFQATAQGTAAFIMSVIQGITLI
PVLYIANRMNGFHGVVWSLVIADAVAFLVGAIMLYVLRNKLQPDLDHLVQ
>lcl|NZ_CP026520.1_prot_WP_241688905.1_3425 [locus_tag=PC41400_RS17360] [protein=TetR/AcrR family transcriptional regulator] [protein_id=WP_241688905.1] [location=complement(3818971..3819546)] [gbkey=CDS]
MQSGKLALRSQEWIQTALFQLMEKKEYSHISITEIADRAGLARQTVYRNFQDKDVILLNYLDGFMKGMWHDVGVGMKTTF
DEEMFVLLFRNWQRNAPAALIHSILTKDRKIRQLIYKSICDYFDELLASRRNEPGENGEEAMHTHGLYAGKSLSSIVHMM
LIEWTLNDFHLTPEQIGSVVSDLTASIRAYL
>lcl|NZ_CP026520.1_prot_PC41400_RS30375_3426 [locus_tag=PC41400_RS30375] [protein=diadenylate cyclase] [pseudo=true] [partial=5'] [location=<3819684..3819806] [gbkey=CDS]
HCAALGLTGQRDALVLVVSEQTGRFPAALNGHPSPLRPAK
>lcl|NZ_CP026520.1_prot_WP_042226420.1_3427 [locus_tag=PC41400_RS17370] [protein=iron-siderophore ABC transporter substrate-binding protein] [protein_id=WP_042226420.1] [location=3820047..3821015] [gbkey=CDS]
MGKSKKWMALPLLLIMSLLVAACAKPNASQPPEEKSSGAPVRMIKHAMGDTPVTGTPQRVVVLTNQGTESLLELGIKPVA
AVKSWIGEPWFDHIKDKMDGVKVIGDETQPNLELIASLKPDIILGTKVRHEKIYPQLSAIAPTVFTENLGDSMIENFELV
AKALNKEAEGKAALDNYNKAIEQTKAALGDKTKMRVSLARFQPGKVRVYYKNNFAGVVLEQLGFARPEAQNKDKFTDDLT
TEQMNVLDGDVFFYFTSDRKGETDASKTADEWLKSPLGQNLNVVKKQRTFNVNEAVWNTSGGIIAAKLLVEDIKVRFEKL
SL
>lcl|NZ_CP026520.1_prot_WP_042226422.1_3428 [locus_tag=PC41400_RS17375] [protein=monooxygenase] [protein_id=WP_042226422.1] [location=complement(3821182..3822684)] [gbkey=CDS]
MEHAYEVIVIGGGPVGMMLAGELALAGIKVCVVERLKETTPFSRALTVHPRTLEILDMRGVKSKLMNKGRPLPRGHFAEL
DTPLNFSVLDSTSNYTLFLPQSETEKVLEEWALGLGAEIRREVEALSVHQDEDGVDVQLAGPDGETALRALYVVGTDGAG
SLVRKQAGISFEGTDATFTAILGDAVLAELPSKSVVSRISEKGLVRVMPINEHLYRVLIIDPERSHVSKNEPLTVEEFRA
SLIRTAGSDFGLTEARWLSRFGNATRQAGRYRNNRLFLAGDAAHIHFPAGGQGMNVGLQEAMNLGWKLAGAIKGWAPDWM
LDSYHTERFPWNTALLRNTEVQTLLLDAAPPIMEMRSMLSKLLSIPEANYQISAQIAAIDVHYAPDAEAPPHPLNGKRSK
DIALRLKDEQRVSMYQLLHKGTFLLLQLNCNEQSEQNREEYRNLQVVSASLDVTGDDWNDVHSALIRPDGHIAWAISRSN
SNPRQTIKEGVARWCGNVTE
>lcl|NZ_CP026520.1_prot_WP_042226424.1_3429 [locus_tag=PC41400_RS17380] [protein=RidA family protein] [protein_id=WP_042226424.1] [location=complement(3822709..3823110)] [gbkey=CDS]
MELLNPKTLTDKVTSVISHVVVTTASKLAFISGQVAVDETGKLMGSGDYRTQAIQAFTNLKYALDAAGADPLQMAKMNIF
VVNHRPELVSVIFDAGLQVFGSNWPKTASTYIGVQSLADPAWLIEVDAIVTFP
>lcl|NZ_CP026520.1_prot_WP_042226426.1_3430 [locus_tag=PC41400_RS17385] [protein=MarR family transcriptional regulator] [protein_id=WP_042226426.1] [location=3823407..3823883] [gbkey=CDS]
MDAIKTNKAAIHESFIGILQQKEQYESRITGSHLEEIRKIIGAEFSLNMTELHTIACIGEQEPINVTSIAAMLDLSKGNT
SKVTNKLLKAGWVRKAQLNDNKKEVYFRLTPVGKKLFALHDELHAKEAQRMNEFLGKYTEAELEFTKRLFGDLVDFYR
>lcl|NZ_CP026520.1_prot_WP_042226428.1_3431 [locus_tag=PC41400_RS17390] [protein=hypothetical protein] [protein_id=WP_042226428.1] [location=complement(3824227..3824712)] [gbkey=CDS]
MKKNFSKLVASATILSTALLISVAPASAATTLFAGDSSAVSVNDGLKSIGITPYAIKDTLTWDLTGNVVDPSKSYSVTGG
YPNVKLYAKNTGSKDFRIEVKHDTKNKVIFDKTVEAGKSVEVINNDSSPTVPSGNYTVTIYGGSGLPKGQVVLKSSDTPW
Q
>lcl|NZ_CP026520.1_prot_WP_241688684.1_3432 [locus_tag=PC41400_RS17395] [protein=DUF998 domain-containing protein] [protein_id=WP_241688684.1] [location=complement(3825338..3825940)] [gbkey=CDS]
MTDLQTLVAKPPVTAKPVTVTASRFALVSVFSGTWFILVLCSLHLLEPEFDPTWRFISEYALGGFGWMMRLAFGLLAAVQ
ICVAVAIYPQIRTVTGYIGLVILGISAIGVLIAAIFITDPISISPDGATFSGRMHSIGAMLDYTPVAALLLSISLTRDNT
WRPVRNRLFASAGIALAAMIVFVLQISLVPKYWQDCLDES
>lcl|NZ_CP026520.1_prot_WP_268640901.1_3433 [locus_tag=PC41400_RS30380] [protein=alpha/beta fold hydrolase] [protein_id=WP_268640901.1] [location=complement(3825970..3826941)] [gbkey=CDS]
MFTTVLFEEGIGGTRISIVTRFATSEQLDNAEAMGMIEGFKDAYDRLEEYLETLMGGPMNTVISKDGTRIAYKKQGNGPA
LILVSSAVADHQDAAQLAEQLTSHFTVYNYDRRGRGHSTDTAPYSVEREVEDIAALTEVAGGQASLFGSSSGAVLALEAA
SLLGDRVTKLYLYEPPFIINDSRKPVPVDYVQQLNSLTEAGKRSDAVEYFASEALGIPGEFIGYMKADPSWNKMEDLSHT
LAYDGLIMGTTQSGKPLPTNRWVVNAPTLIMTGENSEPLFHDAAQALVKLLPKAAIHTLAGQDHSAVITAPEVLAKTVVD
FEL
>lcl|NZ_CP026520.1_prot_WP_255246820.1_3434 [locus_tag=PC41400_RS30605] [protein=SRPBCC domain-containing protein] [protein_id=WP_255246820.1] [location=complement(3826884..3827273)] [gbkey=CDS]
MRRKTIIRKNRERRELTVERTVAIPTKLAWEGWTQPEHITRWWGPKNWTTTVYEMDVRPGGVWRYSLKSDDNSGEEAYCR
AVYEEVEAPRKLVYTETLLLPMTGISFPIARCSRQSCSRKELEVRGLAS
>lcl|NZ_CP026520.1_prot_WP_164977138.1_3435 [locus_tag=PC41400_RS29865] [protein=hypothetical protein] [protein_id=WP_164977138.1] [location=complement(3827233..3827379)] [gbkey=CDS]
MKGLNSFIASCTLNDPKIWNERFDALEIYLKRMNTDEKKNNHSQKPRA
>lcl|NZ_CP026520.1_prot_WP_042226433.1_3436 [locus_tag=PC41400_RS17405] [protein=metalloregulator ArsR/SmtB family transcription factor] [protein_id=WP_042226433.1] [location=complement(3827376..3827711)] [gbkey=CDS]
MNTSVMQLTLLTLAEPNRFNIVELLKKGPRSVGEIVQALGISQPHVSRHLRILSEAGLVRSRSKAQQRIYSLEAGPFQEL
DEWFDSFSILWEERLDNFEDYMLDYKKKEEQ
>lcl|NZ_CP026520.1_prot_WP_042226435.1_3437 [locus_tag=PC41400_RS17410] [protein=nitroreductase family protein] [protein_id=WP_042226435.1] [location=complement(3827877..3828539)] [gbkey=CDS]
MSVSQPSLTVEQQQFFDVIKDRRSVRSYDPEVKISREEMNEILQQATLAPSSANLQPWRFLVIDSPELKQKLFPIAYNQQ
QVIEASAVIAVLGDLESIKLAEKIYGMAVDAGHMPAETAKSFVERYQGIFSSMPPETIRRIVSIDGGLISMQLMLVARAK
GYDTVPMAGYDEAKFVEAFNISERYAPVMLIAIGKAAKPGHPTLRLPIEDVTFYNELPKA
>lcl|NZ_CP026520.1_prot_WP_042226436.1_3438 [locus_tag=PC41400_RS17415] [protein=Rrf2 family transcriptional regulator] [protein_id=WP_042226436.1] [location=complement(3828673..3829086)] [gbkey=CDS]
MISNRGTQIGPPRFGVAVHALILLAQCEGMLSSAAIAGCVQSHATFLRRVLAQLTHTGIVEAREGRGGGYYLKRRPDQIT
LADIFMAVKSDCCEQGVEPDCATGLQIDTKLEAIMGEVQKQTVELLKQHTLADLMQG
>lcl|NZ_CP026520.1_prot_WP_042226438.1_3439 [locus_tag=PC41400_RS17420] [protein=NAD(P)/FAD-dependent oxidoreductase] [protein_id=WP_042226438.1] [location=complement(3829259..3830170)] [gbkey=CDS]
MNKVDVAIIGGGPAGLSAGLMLGRARKRTIVIDEGRPRNGVTREAHGFLTRDGISPSEFRRIAQEQLRVYPSVSQAEDTV
VSISGEDGSFLLETATGMKIASKKLLFAAGMKDHKLEIPGLAEVYGKSAFVCPYCDGWELRDEPLVVISRGAALLHFAPL
LSGWTKRFVVCTNGPDELSEAERDELRDHGIPLFDTPIRAILSSEGMVNHVVLEDGIEIPCTGIFFKSDLVPGTDLPQSL
GCRTSDTGVIEVDKFGKSSVPGVYAAGDTSSLMHQSIAAAASGAAAGAAINGELNQEAWRKTQ
>lcl|NZ_CP026520.1_prot_WP_042226439.1_3440 [locus_tag=PC41400_RS17425] [protein=class I SAM-dependent methyltransferase] [protein_id=WP_042226439.1] [location=complement(3830224..3831123)] [gbkey=CDS]
MRDELFDSAVWEKAWKEDPYTGVNKMKRAGIDPAHSLDSAAESFNKEVFSEEGRRRARRIMNWLEDQGVEFSGSSVLDIG
AASGGFSVPFAERGADVTAVETSRPLIELLEQNSSGLTNGTVKVVHEPFENIDLEVRGWERAFDLVFVSMCPVLSDWASV
EKVLSCAKAFCYMSLSVGSRKHSLVDEVWPLVTDRPRETEHLEMIYLTQLLLLKGYSYQSLVTEEMKITTVSKETAFKET
INWLKMYGIAADDRVRDIVSRYLDTRYPGGQVEIRQGGRFGKVLVRLQNQHMYDREDRI
>lcl|NZ_CP026520.1_prot_WP_042226440.1_3441 [locus_tag=PC41400_RS17430] [protein=MDR family MFS transporter] [protein_id=WP_042226440.1] [location=complement(3831263..3832717)] [gbkey=CDS]
MQAVTTEAQHSAVQKYKVIPIMVSLLLSGFIGMFNETALNVALSDLINLFHITAATAQWLTTGYLLTLGILVPVSGLFLQ
WFSTRQLFTASLLLSLLGTAVATVAPGFELLMLARVIQAAGVALILPLMFHTVLVIVPPEKRGMTYGLIGLVLVVAPAVG
PTLSGLLIEQLSYNWIFGLSLPVLAFSLLFGMRYMQNISIITKPKIDIYSLILSTLGFGGIVFALSSAGEGEGGLGSPKV
VIPLIVGILALVIFAVRQLRMKQPVMDLKAFKQPMFTVGTVMLMIAVMVMMAAMLMLPMYLIRGLGLNAFSTGLVLLPGG
IVNAVMQPIMGRLFDKFGPRWVVPAGMALVSVVLWFLSGITTTTTIPQIIALHTFLMIGISMVWMPSQANGLNQLPPSLY
PHGSAIMNTVQQVAGAIGTAVAVSIMSVGMSSFLKQTADQTDPANVTLALTAGIQNVFEIAIVVAIVGFVFSLFMKRVHI
PEQK
>lcl|NZ_CP026520.1_prot_WP_042226441.1_3442 [locus_tag=PC41400_RS17435] [protein=DinB family protein] [protein_id=WP_042226441.1] [location=complement(3832822..3833304)] [gbkey=CDS]
MDVLKSQYEMIQRTRESLFRYCETFSKTDYVKEVEALGGDSIRSLQAHVADCYRVWLGSRALGKSLPKIKPESIDNVQEM
REIFKKTDALVHEFLSEFKNKWDPTVQASWQSDSVELTELWLFTHTTTHEFHHRGQMLKMGRQLGYIPPKMNLAKLKRFH
>lcl|NZ_CP026520.1_prot_WP_042226443.1_3443 [locus_tag=PC41400_RS17440] [protein=ThuA domain-containing protein] [protein_id=WP_042226443.1] [location=complement(3833362..3833994)] [gbkey=CDS]
MKAIALGQYEGAKYHPFTYIDHELQAIFEQAMDVECTDDYAVLQEDKLKGYDLFISYTDFSDLEIPEDQVQALLSYVKNG
GGLLAIHNGISLQQSEELEKLIGGRVTGHPEYTSLPIRYNHGSHPIVGNLSDFAIDEEPFRFELSPHDSAVILAEYDHDG
QRWPAAWAHTYGSGRVVYLMPGHHLPSFRVESVRKLIRNSGMWAAQRFVE
>lcl|NZ_CP026520.1_prot_WP_042226445.1_3444 [locus_tag=PC41400_RS17445] [protein=Rrf2 family transcriptional regulator] [protein_id=WP_042226445.1] [location=complement(3834167..3834568)] [gbkey=CDS]
MKYSKATDYALHTMLFLTAATPQKPIGVQQLAERQSVSPTYLSKILTKLVKAGMIESITGVNGGYRLKRNWENISFLDII
HAIEGSASLFDCSFEHGPGCPIQKVVLLAEEKMEEQLRNQKMSDLVREYNIVI
>lcl|NZ_CP026520.1_prot_WP_042226447.1_3445 [locus_tag=PC41400_RS17450] [protein=VOC family protein] [protein_id=WP_042226447.1] [location=complement(3834699..3835046)] [gbkey=CDS]
MIKGFGGIFWRTKNMDVIKKWYSEVLQIEIENWNGTVIKPQSGNETIFSFFTENDNYFPAEQQVMLNFEVHNLDETIKHL
EHIGVPLAKNKEVSEFGKFIWIEDPEGRLVELWEK
>lcl|NZ_CP026520.1_prot_WP_042226448.1_3446 [locus_tag=PC41400_RS17455] [protein=arginase family protein] [protein_id=WP_042226448.1] [location=3835272..3836129] [gbkey=CDS]
MTKKTIRLLMPQWQGGNNPNYSFGAELLAWLAPDNDQPLIHVPVQAYDGIPLENENGTYGRKQLLEQLEAAHHIIDAYKP
DRIVMFGGDCLVEQAPFAYLNERYKGELGLVWMDAHSDLGRIVGYDYGHTLPLGNLMGEGDEEFAKHVKIPLNPENVFIA
GINVQTDLDTEIIQRLGIKTAGTQELMNGTKSIKEWIKQSNIKHLAIHLDLDVLDPKAFRSLLFANPETPYEFSAAGTMQ
MSGLLNLIKELSEETDVVGLGITEHLPWDSINLRKLLGEIPILNK
>lcl|NZ_CP026520.1_prot_WP_042226450.1_3447 [locus_tag=PC41400_RS17460] [protein=AAA family ATPase] [protein_id=WP_042226450.1] [location=complement(3836293..3836823)] [gbkey=CDS]
MLYIFSGLPGTGKSTLSSALARELQAAYLRVDVVEQAMRVAGTWDDGPAGYIVCYAIASQNLRLGLDVIADTVNPIHETR
QAWRSVAESLETPFVEIEVICSDDREHRHRVETRAADIPGFVLPTWEKVRNRQYDVWDRDHIIIDTAHQTAAESLMTLRE
LLERERRKFNGLSERT
>lcl|NZ_CP026520.1_prot_WP_042226452.1_3448 [locus_tag=PC41400_RS17465] [protein=CPBP family intramembrane metalloprotease] [protein_id=WP_042226452.1] [location=complement(3836913..3837626)] [gbkey=CDS]
MLGMIVMVLLSWLLLRFTIRQPLSVLGYNPIGLRLRQFGIALLIGLVVCAVVKWTESLITSAGWEVNAGYRWTDFFAASW
WSFKSVWFEELLFRGALLVILIRWFGVRPGIVLSAAAFGVYHWFSHGLFGNIPVMIFMFATTFVMGLVWAYAYSESGSMA
IAAGSHLGWNAASAILFSDSPIGDQLFIPFKGAEYVPLTGLPSLLFFIASNTALPFILFLWIKYKIRNRPRGETAFF
>lcl|NZ_CP026520.1_prot_WP_042226454.1_3449 [locus_tag=PC41400_RS17470] [protein=2,3-bisphosphoglycerate-independent phosphoglycerate mutase] [protein_id=WP_042226454.1] [location=complement(3837917..3839155)] [gbkey=CDS]
MLQRKVILAIADGVGDRPHPLLNHETPLEFAKTPHLDKLALMGISGMMDLIGSGIPVGTDMGHMILFGYQPHHYPGRGPI
EALGIGMEVQEGDVVLRCNFATVDDNGIVVDRRAGRIRQGTDEIADAINGLELQGVKVYVKPATEHRAVLILRGAGLSEN
ISDSDPKAPNDGCPYQEVQPLDDSEESKRTASILNNLLSKFHAILSDHPVNAKRVEQGKLPANFLLTRGAGKLARLEPVT
DELNIKGSCIAGESTVLGVARLAGFKAITDSSMTGNIDTNIELKAKLAIEEIAHSDMVLVHFKAPDLKGHDNEPLEKAKA
IEQFDKMVGLIAQGLPDNTYLALAADHSTPCEVGEHTGDPVPVLIYGPSVRRDRNWRYNEIDCAYGGLGHLTGNEFVRTL
YGLMGRVKKLGN
>lcl|NZ_CP026520.1_prot_WP_042226455.1_3450 [locus_tag=PC41400_RS17475] [protein=DASS family sodium-coupled anion symporter] [protein_id=WP_042226455.1] [location=complement(3839173..3840660)] [gbkey=CDS]
MSEIKGNLESDDKQPPIKKKSKMEESMKKWGIPLAITVGLLLFFMPTPEGMSAQAQSSIAIFTGALILWITTPIPIYLTS
IMAILLLPLVGAVKDQSVAFGTLGYDVIWLMVSAFILTAAMIKSNLGRRLSLWMVTKFGKTPKTTLFLLIVINFILAFFV
PSTTARATLMVPICLILLEVYKAIPGKSNLGKTMMLQGVQADAIATSGVMTATSGNIIAVGFINDQAGGNIGYMDWLLAS
MPTAIVTMLITFFVGLKLFSFKDETNFTSAMDTLKDELKKLGSFSLDEKKAVAIFIFTVVLWATGDYHKAWFGFEISTEQ
TAVLSALLCLLPRVGLLTWKETNIKWELMIFAAGAYAVGNALDKSKGAEWIIGKVVDALGLEQMNHALVYIVVVFLSMYS
HLIFTSKTVRVTILIPAFIALAKTLGLDPVPLALAAAMTMTYTITLPPHSKVNTIYFGTGYFTVLDQVKYAVVTCFIGAC
VISLAIFTWFKVIGI
>lcl|NZ_CP026520.1_prot_WP_042226457.1_3451 [locus_tag=PC41400_RS17480] [protein=LysR family transcriptional regulator] [protein_id=WP_042226457.1] [location=3840907..3841809] [gbkey=CDS]
MNFSKLQTFLTLSECLNFTEAAELLYCSQPSVSMQIQSMEAELGVLLFDRIGKKLYLTKQGEEFKPYAEQMINLLHSAKD
HIRQLEDLSYGTLSFGASNFVGVYLLPMILRNYSKTFPNIKINMSITSSSHLIHMLESNKVEFLVLSDRVLMDESRFERK
TFYEDELVLIVNPLHPLAAKPVCTLQDLANETFIIKPDKSATRGYLEGKFEELGFTPPKYLEISNLEGIKQAVIHELGVS
IVSNFAIQQEIKSGLIVKIPVKGVTFTRGICYVYHRQKHFSPAAKQLVSLLDQFYDVNSN
>lcl|NZ_CP026520.1_prot_WP_053228621.1_3452 [locus_tag=PC41400_RS17485] [protein=response regulator transcription factor] [protein_id=WP_053228621.1] [location=3842272..3842907] [gbkey=CDS]
MHLEQAELQVIEAGSGRQAIQYLQEYAIELVILDLMMDDGNGFEVLQYLRDANMESLVIALSARREVQDKILTLGLGADD
YVTKPFSPVELVARVQAQLRRHRPHSSYPAKILRLNKLVLDIENYLLRNDGISFSVTPVECRLLEMFMQNPDRVLTKREI
YKQVWNHENFDDNNLSVFISKLRSILEPTEDSPHYLHTIRGIGYRFSGDGQ
>lcl|NZ_CP026520.1_prot_WP_042226460.1_3453 [locus_tag=PC41400_RS17490] [protein=HAMP domain-containing sensor histidine kinase] [protein_id=WP_042226460.1] [location=3842904..3844328] [gbkey=CDS]
MKNRTKLWLLMLISAITSILLFSSLSLVMGKIGDRGYDLKSLNALSQEVLNTIATHQAFQIEDITPILDNAHSNHPDIRY
EWISADGSTIYDTFGEKKTYDFEQLANRMLNMPQNLWGVDEPVTLTFSISQAHQPYYLLMSLSSDDMKNGQVYFFMRTFK
ALSTFMLPLIVALLIPYLLSLWFFSSMDKRINKLNHALGQLNLQSELTVLEDTKKDEIGQLTTHYNAMARRIQNQGEQIK
QFDARRNLLLSNLSHDLRTPLTMILGYAETIRAGLYKDEKELQASAKVLLQRSRYIDKLLDQLLDITRQDEGNLELHVDL
HNVSEMMRKVAADYLMFLEGQNFTVEVNIPDEDIQAWIDASLIERALRNLLDNAIRYGSEGHYLEVGLTEKYDALFMTVI
DKGRGIPLQDQKHVFERFYRVDGSRKGEGLGIGLSIVQEIISLHDGSITFTSVPFERTVFEVQLPNHQREQSKS
>lcl|NZ_CP026520.1_prot_WP_084706501.1_3454 [locus_tag=PC41400_RS17495] [protein=fibronectin type III domain-containing protein] [protein_id=WP_084706501.1] [location=complement(3844466..3847618)] [gbkey=CDS]
MMRSNLLEIRKLIICLLTLALITLGSWTGGSIQEAKAAVSGDYEYTESNSEVTITGYTGTDTEVDIPATIDGYPVTSIGG
YAAFYNKQLTSVTIPDSVTSIGDAAFYNNQLTSVTIPDSVKSIGVSAFHWNKLTSVTIPNSVTSIGNNAFYWNELASVTI
PNSVTSLGDFAFSDNYGLTSVIIEGDSTAIGTNAFSNSFKPVSVIAFDPSIAKTYASDWGHTFLNILSYGVSYSPNGESE
GKQANSTKVTVTVPDFVGSSKTYYRWSTAVTVPDWPNVEWVEFTSEESISTPSAVGTWYLHIRVDDQNGSHAAGYSHSNA
FKVVAVPSAPTNVTATAGNGQATVNFDAPLSDGGSAITLYTVTSSPGGFKGTSTTASPVMVTGLTNGTAYTFTVVAKNSA
GNSAVSAASGSVTPVAPISVPSAPTNVTATAGNGQATVSFDAPASDGGSAITGYTVTSSPGGFTGTSTTASPVTVTGLTN
GTAYTFTVVAKNSAGNSAASAASGSVTPVAPISVPSAPMNVTASAGNGQATVNFDTPLSDGGSAITGYTVTSSPGGFTGT
STTASPVTVTGLTNGTAYTFTVVATNGAGNSAASLATSSVTPVAPIGVPSAPTNVQATAGNGQATVSFDAPASDGGSAIT
LYTVTSSPGGFTGTSTTLSPVTVTGLTNGTEYTFTVVSTNGAGSSPTSSPSNPVTPRNSSSGSSSGSSSSNHGSISSPIY
STNGKLTLPLGSTGEVSLDQEIFIRIPANATSKPLEINIEKVSNTQNLFAHNEISASPVFDVQKNIPENLSKPATLTMVF
KQSQVNSNQTVAIFHYNEATKTWVKIENGKIEGNRISVEVDNFMKFAVLVVDRASGLPIGAPSIDESKEVNFSDIAGHWA
VSDINKAVQSGFVNGYGDGTFKPNAIITRAEFVVMLMKALKRQEEAAKLTFNDSAKIGTWAQKAIEQAVQAGIVNGYSDG
SFRPDAEITRTEMAVMIAKALALKVEVNTITDFVDDADIPSWGKAQVAVLKNLSLISGQGSNKFAPHDKATRAEAVTILL
KMIDQKSKQS
>lcl|NZ_CP026520.1_prot_WP_053228625.1_3455 [locus_tag=PC41400_RS17500] [protein=penicillin-binding transpeptidase domain-containing protein] [protein_id=WP_053228625.1] [location=complement(3848116..3848388)] [gbkey=CDS]
MKKWDGNRRAREIWNQDHTSASAMRYSVVWYYQAMARDIGKERMQEWVNRADYGSKDFSGGIDRLWLNSSLKISPVEEVD
FLADHHPGRE
>lcl|NZ_CP026520.1_prot_WP_042226462.1_3456 [locus_tag=PC41400_RS17505] [protein=ATP-binding protein] [protein_id=WP_042226462.1] [location=complement(3848526..3851690)] [gbkey=CDS]
MANHSSKYQIKKSNMLVMLGLLLIILSGSRILWMEMFHDQKHEPIKNGQFDLRNWNAEDGEIVLLDGEWEFYPSQRLMDG
RQLQGFGESKPRLIGVPGRWNEALHAGKSTPYGFGSYRLRVYVNPEKDLNYSIRVPSVRTASELYVNGRLLTKSGQVATI
KDEYISKDLPYSTTFTADDNGVIELVIQAANYVDSRSGGIVRSIKFGSEEAIAKEMKFSIFMQFLAAIIFIMHAVYALIL
YLLGNKEKKLLYFSLLALCVTITSVFSTDEKLFHQLFYIGSNWDFRLTNFALVIAAYALLQCTNHRELPYWRRMYPVYKI
MSLGTASITLFLNPTQVIMLFPIYYVLGGTAAVITLITMFKKVIKDLKSNLLLLFSFLALMNHSLWSLILLDSGLSLVYY
PFDMIIAMACLASVWFKDYLIMHANTKELATTLQRMNDHKDQFLANTSHEFKNPLHSILNMSQSVLKRERHLLQERSIKE
LETVLSVGRRMTLILNDLIDVMSLREGNPRIQQRVLFIQPIVTGVLDMLQLNAEVKSLQIVNHIPEDFPPVHADENRVTQ
IIFNLLHNAVKYTNDGVISIRAYVEEGQACIVIADTGIGMDENMLKRLFRPYEQASTRETMIEGGFGLGLSISKQLVELQ
GGTLAVSSVAGEGSEFKFSLKLADLVAEEENDRTSALEPLPLRSVPIQENVWENGEKADIPPVVKPTILLDVHRDRPHLL
IVDDDPINLQVLETILPPEDYEVTMVTSGKEALAVLGTKEWDLIISDIMMPQMSGYELTRIIRERYTLAELPILLLTARS
QPQDIQSGFLAGANDYVTKPVEAIEIKSRIEALTTIKQVIREQLQLEAAWLQAQIQPHFLFNALNSVIALKEINLDKMGN
LLHELSNFLRNKFKFENPNGLVPIEEELSLVRSYLYIEKIRFEEKLQINWEIDDYAELEIPFLTIQPLVENAIRHGVMNR
PRGGKIVIRVSVYDTHAEITIEDDGIGMDKDQLQRILERRADSSSGVGLINTDQRLKRHFGTGLDIQSTIDNGTKVTFRV
RNRSNSADTINPLS
>lcl|NZ_CP026520.1_prot_WP_053228626.1_3457 [locus_tag=PC41400_RS17510] [protein=zinc-binding alcohol dehydrogenase] [protein_id=WP_053228626.1] [location=complement(3851885..3852892)] [gbkey=CDS]
METLSIVFAEKQKVEIRKESFDSKLGKTEILCAAITSLISTGTELQCLRGVFDKDTNWASWVKYPFYPGYSMVAEVLEVG
EDVRAIHKGDRVFVDGPHTQYFKTEATHVTVLPDEVSSEEGVWISLAKTTQLGVRRAELFLGETVGVIGLGLLGQLVTQY
LYLSGAKEIYAIDPAESRIRLLPEKAGIHRMAMDGAKAKAEISKKTKGKMLDVVFDITGHPAVLAEATQLVNKFGRVILL
GDTATPTAQGIGSNVVSNSVAILGIHGTMNYKGWDHPDMAALYLSYITQGRMNVAPLITHRYSPLEAAQVYDSLVQNRSS
AMGVLFDWTNLKDQS
>lcl|NZ_CP026520.1_prot_WP_042226464.1_3458 [locus_tag=PC41400_RS17515] [protein=AraC family transcriptional regulator] [protein_id=WP_042226464.1] [location=complement(3853017..3853850)] [gbkey=CDS]
MKDDILSSLLETAELKILTFDIHHRTEMNYRKNIIKEFYMMTYIRKGSAKLRVQNQIYDIDPGTVIFIPPDLEYDQYKDN
DEETEFLWWNFTFKIQNVVDVLPLLQIPYIYPLKNHKKFEAVFDEFMLSRTQGGYISNIILQKAKSLELLYLLVDNAFEK
VHVAEMRQQSHSFLSVLLKIVSQPEASFSLSDLAKELHMNPTYVSNRFKEMFGKSPILLHREIKIEKAKSLLKIQEMNMT
EISTSLGFNDVQTFTRLFKKYTGISPSQYKGLFDKNL
>lcl|NZ_CP026520.1_prot_WP_042226465.1_3459 [locus_tag=PC41400_RS17520] [protein=bile acid:sodium symporter] [protein_id=WP_042226465.1] [location=3854013..3854990] [gbkey=CDS]
MISREQLEKNQVWLYAFVLIIAAGFGLLWPDSSENLDRMISFVLAILMYGMFAQIPFFRLKEALANRKFIYALLITNYLV
VPVLVWGLTRFLPNDPAILLGVLLVLLTPCIDYVIVFTHLGRGNEKLILVSTPILFVTQMLLLPVYLWLFMGKTAAEIVS
VGPFLEAFLGLIVVPLVLALLTQWWSRRQPLGERFLDATAWLPVPFMALTLFVVVASQISKLYTSFDSIIQVIPVYAAYM
AIIPIIARIVSRIFRLDTGAGRALIFSAGTRNSLVVLPFALALPEPLNGVASAVIVTQTIVELIGELLYIRLVPKVLLRD
TEVPN
>lcl|NZ_CP026520.1_prot_WP_042226468.1_3460 [locus_tag=PC41400_RS17525] [protein=DUF6366 family protein] [protein_id=WP_042226468.1] [location=complement(3855084..3855299)] [gbkey=CDS]
MSKEKETPERRRERLRHEELKRNPTGNMSDAFNRAENGNLTDLTGSLGWKSTGILILVIIIGIVVASIVLK
>lcl|NZ_CP026520.1_prot_WP_042226470.1_3461 [locus_tag=PC41400_RS17530] [protein=chromosome condensation regulator] [protein_id=WP_042226470.1] [location=complement(3855430..3856326)] [gbkey=CDS]
MDSISPIEAALQVKRWPKETLAAGRRHTVGLKSDGTVTAVGDNKCGQCDVSGWRDMVAVAAGNVHMATNTGNAHTIGLSS
DGTVAAVGWNKHNQCDVNDWRDIVAVAAGWRRTVALKSDGTVVAAGRNNEGQCHVSGWRDIAAVAAGDWHTAGLKSDGTV
TIVGNNRYGQCDVNDWRDMVAVAAGYLHTVGLKSDGTVAAVGLNKHSQCDVSGWRGIVAVAAGSYHTIGLKADGTVVAAG
RNEHGQCDVSGWRDMVAVAAGCAHTVGLKSDGTMVAVGDNEYGQCDVSGWRGIRLPID
>lcl|NZ_CP026520.1_prot_WP_042226472.1_3462 [gene=cysC] [locus_tag=PC41400_RS17535] [protein=adenylyl-sulfate kinase] [protein_id=WP_042226472.1] [location=complement(3856800..3857363)] [gbkey=CDS]
MIEKSEWNRVNGHKGKVLWFSGLSGSGKSTLAAETERELFRRGIRCVVLDGDQLRSGINRDLGFTDQDRNENLRRAAEIA
ALFLSVGFVVLVPMISPSAETRNTIRQRFGPEDYTEVYVKCSLETCEQRDPKGLYRKARSGLIKQFTGIDSLYEPPLEAE
LTVDTEHEPTEHCVQLLVDSILCDCQL
>lcl|NZ_CP026520.1_prot_WP_084706502.1_3463 [locus_tag=PC41400_RS17540] [protein=sulfotransferase domain-containing protein] [protein_id=WP_084706502.1] [location=complement(3857411..3859207)] [gbkey=CDS]
MKNAIELHGVSKKFNKLSGQRNTIKDIVTTPTPAASKEKWVLRDIELEIPKGTSLALIGKNGSGKSTLLKLISRILHPTS
GRVKVEGKVATLLELGAGFHADFTGRENIYFNGSVLGFTRKEIQEKMDSIIAFSELGEDIDRLVRSYSTGMYMRLGFSVA
IHMTPDILLIDEVLAVGDHEFQKKCINAILRLKSAGTTIVFVSHAGEQAKIICDLAIWLEDGRIKLSGPTRDVVMAYEQS
AAAHDYLAEYYYRMGCRSQQQKDFEKALQFFDQSLDKGYNEFSVKILRSSVYLELGMQEEASEDAVRCLETAPQGTDMES
VVQHVHFINSKRHVEWVRSARTKLFHNTTTEPAFLIIGTQKGGTTSLYHDLTQNPNIKSAAVKEVHFFDEQYHFGYDWYK
KNFPPDLPEGSITGEASPYYLFHPQTPGRVREWLPDVKLIALLRNPIHRAYSHYQMMVRRGLEPMSFPEALQAERSRVEG
EYDRMSRDPRFTSQNCSIYSYLKRGLYAEQLERWYRYFPKDQILVMSSETLFRNPAESYRKVLAFLDMPLWEPGGYASLN
EGENDRLIPKDTMEWLRNYYTEPNRRLFELIGEAYAWD
>lcl|NZ_CP026520.1_prot_WP_164977139.1_3464 [locus_tag=PC41400_RS29870] [protein=acyltransferase family protein] [protein_id=WP_164977139.1] [location=complement(3859233..3861074)] [gbkey=CDS]
MARSRWIYTDVLRIAAMIGVVLIHITAPIIGIRSDIGDIHYWWTAHIINAAFRWSVPVFVMISGALLLAHGRNESAMSFY
RKRLGKLLPLFLLWSMIYYLWDLRKEMHSFNWDQLATGLVKGTLHYHLWYVYMIILLYLITPLLRVLVRRVPPRAVFYIG
GAGLAFSNLATLAGWKSWSFVIDASSLPGYIGYFLLGYVLSRVELDKGKRRLLYGFGLLSFMAIAGGSHWILSTYPSANM
YFYKYTSVPVLFTAAAVFVGVKQFGFRKATTGGPRLLSFMSSSVFHIYLSHVLIMEWLYERRPWDIIHGNPLGYVPIVAV
VIMSLSLLINLAWLGSLYALKEIFKLAVRIFEERNSIHPLKDIYQYREMLRNMILKDLRTRYKGSALGFLWTFMNPLLML
GVYAAVFSFIMKADIPHFPLFILVALLPWNFFSQSISGGARSLVNHAELMKKVYFPREVIPLSVIGSNLVNYLLTLIILI
PALWLSGIPLTATLSAFPIILLAQTLLILPIVMLSALGTVYLRDMEHIVNVLMVVWFYLTPVLFPLSLIPGSFRWVFDYN
PMTPIIDAYRDIFLYGQWPDFGILLPMLFVLTVVNMGILAVFSLLQKHVVEEV
>lcl|NZ_CP026520.1_prot_WP_042226474.1_3465 [locus_tag=PC41400_RS17550] [protein=sulfotransferase] [protein_id=WP_042226474.1] [location=complement(3861190..3864186)] [gbkey=CDS]
MSAKPIAVIILGMHRSGTSLLAQTVRSLGVHMGHEHQMVPPREDNPEGFWEHAEIVAVHDELLERLSSSWDATKPLPEQW
WLTEGGMASRARLVDIVSRDFSGHPVWGFKDPRTCLLLPLWQSVFEELNMEPRYILSLRNPLNVAASLYKRDHFLHDKSF
AIWNLYVLSSLYYTSHKRRIVIDYDRLLEQPVETGQRISDFLNIPYGAAERARVGGLPKPGLRHSSFSMQDVWEDRLVPA
MVRQTYQIGLSGERDSDALCTDELRENVQNAYDQMVQIDRLLNHEKSYRIRIYWAGADGRFTEDKSTYITVSAKGQKSLQ
TVEIRESLGSVLRIDFFTEPAYFKLSKLSLVYEGEEANLLRGYAKEYVQFLAFTGVGEEADCVSGISTGNTSQLLIRDLP
PHGETATLRLELSCSFEIGEDVVQALQLQHSAEVALSAALHDAGLRIERLLTANAEKEKALKLAAKQIGEQIDTLELLCE
DSRRREAKLAAISRDLQEQRAEASRLNAEIERREQQAKEERVLLIRQKDLEIENYRQEVASYSGSLSWKITLPLRMGREI
ERSAKRRFKGWAKGLLQRYAIRRNQHSLHAGLTGRLPYKQYALIFSHTNYLESMGGTEKYIYEQASYLSQNNVGIIQIYP
GQSYSLLNTKEDTFYGVVVSDQFRGFYSVQEITEWLSRIHRHLAEIYTHHLLNWQISDYTALLRNVEEAGVLPHTFFMHD
FFALCSSYHMMYQPKIRQSIKEAKERRTCIADIITTAGTDAAICQNCTHGIALAEWREALRPVLHQSDRIVVPSDFVKET
VAAVYPDMTDKIRVRGHLIFDRQTIIYKSPPENRKIRIAFLGYKMENKGWSVWERLYKDEALGRLYEFHHIGSQEKYADQ
VSVHRYSFLLEGRMGAPNLLIAQDIDLVLLWSIVPESYSYTLQESIAAGVPVLTSPQSGNIAVTIDAHPELGKVLSGEDE
LKAFLFDPEHVRNYVSGDRARYVLEYNHLPSGKREARS
>lcl|NZ_CP026520.1_prot_WP_042226476.1_3466 [locus_tag=PC41400_RS17555] [protein=DNA alkylation repair protein] [protein_id=WP_042226476.1] [location=complement(3864465..3865172)] [gbkey=CDS]
MNLETVMGELEALGKERTKKIYQANGAHEPLFGVATGAMKPLFRKIKRNQPLAEQLYATGNYDAMYFAGMIADPEAMTEA
DFERWMDKAYFYMLSDYVVAVTLAETDIAQTVSDTWIASDTELRRSAGWSCYCWLLGNRPDREFSEEKMAGMLELVKNTI
HDAPNRTKYAMNNFIYTVGVSYLPLHDKAVAIAKETGPVEVNRDTSKSKFLNASVTIQKAVDKGQIGFKRKYVRC
>lcl|NZ_CP026520.1_prot_WP_042226478.1_3467 [locus_tag=PC41400_RS17560] [protein=LysR family transcriptional regulator] [protein_id=WP_042226478.1] [location=complement(3865214..3866122)] [gbkey=CDS]
MELRQLNTFRTVAFTLNFSRAAEVLNYVPSNVTMQIKALEDELGVRLFDRLGKQLVLTTAGKRFLTHTQEVLNKLDEARS
DVHDTENLSGTLTISANEVLCAYRLPVVFHLFRSRHPGVRLIFRSVPNQELKQTLFEGTADVVFMLDESIRSSGLAVEPM
LEETFRFFAAPDHPLAKRTVLQLEDFHGEVFLTNEKGCPYRTMFDRSFEKEGIDSITYLEFQSAEAIKQCAITGIGIAFL
PEITAKAEVERGELVALPWQIPDLHVYTQMLWHKDKWLSPILLSFIEAAREVIGIEEENKTV
>lcl|NZ_CP026520.1_prot_WP_042226480.1_3468 [locus_tag=PC41400_RS17565] [protein=alpha/beta fold hydrolase] [protein_id=WP_042226480.1] [location=3866266..3867285] [gbkey=CDS]
MDYEIFDLGDVTLQSGVTLPSAFLAYKTYGRLNEKKDNVIVYPTAFGDHHVQNEWLIGNGMALDPQKYFIIVPNLLGNGL
SSSPSNTPAPFDRANFPQVTIYDNVKFQHRLVTEKFGIQKIALVAGWSMGGIQSFQWGASYPDMVERIAPFAGVAKTWPQ
THVVLEGMRAPLMAAVRFDSSKLNELTSEDMRAVGRVYAGWGVSEAFYREELYREMGFDSLADFIVGVWENSFMQMDPHN
VLAMLWTGQHSDISANPDYNGDFDKALRSIKAHACVMPGSTDLFCPADDNEYEAKLMPNAVFNPIESIWGHFAGRGINNA
DNQFIDDNLKRLLALRIDG
>lcl|NZ_CP026520.1_prot_WP_042226481.1_3469 [locus_tag=PC41400_RS17575] [protein=GyrI-like domain-containing protein] [protein_id=WP_042226481.1] [location=complement(3867685..3868164)] [gbkey=CDS]
MEQRNEAGLITKESFRAVGLKWAGTFAEANAGEIRVVHDEMRNRLKEIKHILNPELLLGLSYQINEGGFTLYAVVEVAKV
EDIPEGMTSVTLPTLTYAKCEHKKGQKIDSSYNNMYAWIESQGYKLHKGDVTHFEEYPMHQDPYSKEPEFVIMIPVEIG
>lcl|NZ_CP026520.1_prot_WP_042226482.1_3470 [locus_tag=PC41400_RS17580] [protein=DUF4358 domain-containing protein] [protein_id=WP_042226482.1] [location=complement(3868472..3868966)] [gbkey=CDS]
MLNNLTNRSLKGALLILLLAGLCIAAGCSGKGKSEVIPVDEIGKRIEQSVNLDNLKKGDLEKVQKLYGLEAEVVEDVILY
TATSNVKADEIAVIRLKEEGKSDYVKQRLLKRIDEQTVKFKDYRPDEYALINKHVLKVKGSYVLFAVSKDARLIERAFDE
ALGK
>lcl|NZ_CP026520.1_prot_WP_042226484.1_3471 [locus_tag=PC41400_RS17585] [protein=DHHW family protein] [protein_id=WP_042226484.1] [location=complement(3868971..3870128)] [gbkey=CDS]
MKRFFNTPAASMTFVLLTAIYLIFAVNLLMPSRDFSEAENRSLEQLPSLSLGSLLDGTFTSDFERYISDQFMLRDIWIGI
KSGTDQLLGKKESNGVYIGRDGYLLQKFNPPADSELQKKADTLNGLASMAPGVHTYVMLAPTALSVLTEKVPNYVAGEKM
LEKMEKVRGQLQNVSFVDVYPALHTKRNEEIYYRTDHHWTMRGAFYAYRELGKQMGFVPKEEKDFTIRQVSNAFYGTQYS
ASGYKLVQPDSVNLFLPKVPDDYTVEYTAEKQTSASLYNFEYLSKKDKYALFMNGNHALVKIKTGSQEKRKLLVVKDSYA
NSLVPFLTPHFSEIFVVDPRYYEEDIRKLLDDNQIHDMLVLYNLHTFFEDESLYRMVGPVEKGRS
>lcl|NZ_CP026520.1_prot_WP_042226486.1_3472 [locus_tag=PC41400_RS17590] [protein=MBOAT family O-acyltransferase] [protein_id=WP_042226486.1] [location=complement(3870143..3871549)] [gbkey=CDS]
MVFSSLMFLYLFLPLTLLLYYVSPKPFRNAILLIVSLLFYAWGEPVYIVIMIFSTLFGFMNGLLIEKYRDRKRIARTVFM
GSLLVDLGMLGFFKYAGFVVNNLNEAFGLHMNATDLPLPLGISFYTFQTLSYIIDVYQNKIPAQKNPIAFGAYVSMFPQL
IAGPIVKYGDVAGQLRERKVTLNDFGEGARLFIRGLAKKVLLANNIGLLWTSVKAEPFTELSVLSAWISILAFTLQIYFD
FSGYSDMARGLGRMLGFDFMKNFNYPYISRSVTEFWRRWHISLGSWFREYVYIPLGGNRIGRLKQLRNMGIVWLLTGLWH
GANWNFILWGLYFGFFISLEKLYVLNWLKQKPAFIGHLYTLVIVIVGWVIFENEQLPEAVVFIGILFGIGGHVITDMASM
YALSTNILLLIVCAVCATPLPEKALSAFTKKYKKTGAVIIPGLYALFLLLSTAYLVNATYNPFLYFRF
>lcl|NZ_CP026520.1_prot_WP_084706528.1_3473 [locus_tag=PC41400_RS17595] [protein=GDSL-type esterase/lipase family protein] [protein_id=WP_084706528.1] [location=complement(3871583..3872296)] [gbkey=CDS]
MKPIATTSKTSHTAYKKWAVAAFAGLLVLSLSACGTVNQENGQKTPLTASQSPVPTSSTSSYKTQFKSVVFLGDSITEGL
SFHDVLDKANVLAGAGKTAEFSLMAEDIDDLTGRNPKQVFLQLGSDDILWPTENPKEYSLKHYKTLITKIKEKLPNAKIT
ILTVTPVTEEAEKKEPRYKNIADYNQGLKELAASEKTGFVDLTSLIQYSPDLYDSDGIHFKSGYYTIFLDALKGYVK
>lcl|NZ_CP026520.1_prot_WP_241688906.1_3474 [locus_tag=PC41400_RS17600] [protein=polysaccharide deacetylase family protein] [protein_id=WP_241688906.1] [location=complement(3872356..3873027)] [gbkey=CDS]
MQNEKVLVNEKAAQPAPSTSGKVVYLTFDDGPSELTNQFLDVLKEKGVKSTFFMQGRNLEREKLHSVVKRASLEGHYIGG
HSMTHVYKTLYEEGQFVNEMKQTLSLIRDITGKSPHLVRPPYGSAPGLANKKIRDQLAEAKIKVWDWTIDSNDWNLSGRP
QEILTTIKNTTKRKTEVVLLHEKPQTLKVLPDIIDFYKKQGYTFGVYDDVKHFPLNFTKDERL
>lcl|NZ_CP026520.1_prot_WP_042226488.1_3475 [locus_tag=PC41400_RS17605] [protein=response regulator transcription factor] [protein_id=WP_042226488.1] [location=complement(3873359..3874063)] [gbkey=CDS]
MNKTVLIVEDEPILREIMKDYLTDEGYKVLEAGDGREALDLFHDHEVHLILLDIMLPGLDGWSVCQRIRKNSTVPIIMLT
ARMDEDDTLLGFELGADDYVTKPYSPAILLARAKRLLEGRQANGQASEQTGNLLSSGGITLNFLSRMVTVDGAPCFLTHT
EFELLACLMQNKGNIITRDQLITKIWGYDFVGDERTVNSHIRNLRSKLGDKSACVVTVIRSGYKFEEPQEQGLS
>lcl|NZ_CP026520.1_prot_WP_042226490.1_3476 [locus_tag=PC41400_RS17610] [protein=HAMP domain-containing sensor histidine kinase] [protein_id=WP_042226490.1] [location=complement(3874195..3876021)] [gbkey=CDS]
MRKGIVLKLFLLTTALCMLILAVIFIGQTVFFKQFYVNQKVKDVNAGLLAYKNEYLKNAGDAQTAARLEQQFYQKYNTWY
AVLDSRGGLKNTEDFFMEIRLNHSNNDPAYSGRTIKIPLYKFINVEDFFNGNTFTAPWIKEGESIAFEGLEINNQLVIQR
MGWNPANLREENRLENRQMVTKEYEVVTRYPSPTQYHEKYPNILASGTITNIQIPDGAGNARYMNHLFLERIKAFEANLL
YGDYDPARDANQIIDYEENHVNYKMFVNSFEDRDENPAYLIAMTSLQPVSEAAGVMQHYYVYTALATLLLVLLASFYYSR
RIARPLLRITSMTKQMAGLDFSKKIPVTTKDEIGDLSNNINALSERLHSHILRLNEDIEKEKQLEHTRKEFISGVSHELK
TPLSVIQSCLSILKDGIASHKRDYYFNAMEDEVKRMNLMIADMLELAKYESGTYKMKMGSFYIDTIINQICEKLAVDISD
KQLQLHTSVGHVEVVGSELRIEQVIINFLTNAIRYTPERQSILITTMEEQDKVKICIENKGAHIPDEQLEKIWDRFYRVE
PSRQRSTGGTGLGLAISKKILDMHDVPYGVTNTSDGVIFYFYLNKKEK
>lcl|NZ_CP026520.1_prot_WP_042226491.1_3477 [locus_tag=PC41400_RS17615] [protein=50S ribosomal protein L25] [protein_id=WP_042226491.1] [location=complement(3876399..3876944)] [gbkey=CDS]
MAAYIHTERRPRLNPSGLRNLRQEGRLPGVVFGKNADNEIIHISTKQFEKWLRQGASGFIELKMDGGPSLTVLLEDLHRD
SVTGSLLHVDFQLVQTGEIIRTKLPVKFIGTPVGTKSGGVVQIQDPHVEVEALPKNLPHSVELDIHAMEIGDTCFVKDLN
LPPELTVVSGVNKILVSVSKP
>lcl|NZ_CP026520.1_prot_WP_042226492.1_3478 [locus_tag=PC41400_RS17620] [protein=ABC transporter permease] [protein_id=WP_042226492.1] [location=complement(3877307..3878074)] [gbkey=CDS]
METAKKHFFIDISIMLGRSMRHIFRSMDTIVTVTIMPIAMMLLFVYVFGGAIQSGTDNYVNYLLPGILLMAIASGISYTA
FRLFTDVQRGIFERFHTMPISRSALLWGHVLTSLLSNAISVVVIILAALLMGFRSSAGVLPWLAVAGILALFTLALTWVA
AIAGLSAKTVDGASAFSYPIIFLPFISSAFVPTESMPTAVRVFAENQPVTSIVEAIRALLSGQPLGSDIWIAIAWCVGIM
LLAYVFAMRVYKKRG
>lcl|NZ_CP026520.1_prot_WP_042226493.1_3479 [locus_tag=PC41400_RS17625] [protein=ATP-binding cassette domain-containing protein] [protein_id=WP_042226493.1] [location=complement(3878075..3878845)] [gbkey=CDS]
MQTVQTKAIEVKGLQKSYKQHQVLKGVDFEVEKGSIFALLGSNGAGKTTVVKILTTLLKPDGGTAVINGFDTVFRPDHVR
QAIGLTGQFAAVDEILTGRENLIMIAKLRYLKNPRQVADNMLERFGLTDAADRKASTYSGGMRRRLDIALSLVGNPQIIF
LDEPTTGLDPEARIEVWKIVKELADGGTTVLLTTQYLEEAEQLADRIAILHEGKIIANGTLQDLKKMFPPAETVYVEKQP
TLEEIFLAIVGKQEGK
>lcl|NZ_CP026520.1_prot_WP_042226495.1_3480 [locus_tag=PC41400_RS17630] [protein=pentapeptide repeat-containing protein] [protein_id=WP_042226495.1] [location=complement(3878861..3879727)] [gbkey=CDS]
MNQKLTNYLNSIFTPYDGVKSVTELKADLLSDLQERYTELRAEGKDDETAFKMTVESIGDIEETILEVSNLSRSLERQVV
TNFSASNLAESDFAGVRAHKGQFNGSALHGSDFSGADLTGSSFKGSDVREANFDDANLTDCSLSALDLANASFNKTILVR
TNFSKSGLAGAKFIGVRLTDVSLSMIDLRKTIFEGCVFDGVDFNYSDLSGQCFDGQTFIGVKFEKAALTEVSFKGATFKN
VSFQPGFTFTKKYYRMIKATCFDGATMDKLTYAALKGMEADLSKVTVI
>lcl|NZ_CP026520.1_prot_WP_042226496.1_3481 [locus_tag=PC41400_RS17635] [protein=PadR family transcriptional regulator] [protein_id=WP_042226496.1] [location=complement(3879742..3880071)] [gbkey=CDS]
MSENKITSDLLRGHTDTMILRLLSEADHYGYEIVKLIAERSGGEYELKEATMYSSFRRLEADGDIEWYWGDESQGGRRKY
FRITDKGKDTYARNKSNWEYAKRVLENLL
>lcl|NZ_CP026520.1_prot_WP_042226498.1_3482 [locus_tag=PC41400_RS17640] [protein=glycosyltransferase] [protein_id=WP_042226498.1] [location=complement(3880397..3881608)] [gbkey=CDS]
MARVLAVITPAEGHVNPSLGLVTQLTDNGEEVVYVCTEEYRSRIEQTGAQLLTYPFPQDAFSHDPVMKPQEYKHPYQFIY
MMVGGIIRRMVPEVLRVIGNQKFDYLIFDSLMGWGGTILAEKLGIPAVCSIASFAFVEPLGADQGLNAEETDTKELYEAT
MKITRELAHEFQVSIPAMEEIPAHAGRLKLVYTSRYFQPEAEKLDDSFIFAGPSIIPRQDAPAFPFELLRERYPHTVYIS
MGTILNKNLDFYQLCFEAFGDLPVNVVLSSGRYTDMKPLADRIPANFIVKPYVAQLEMLQHTDVFITHAGMNSTSEALYY
NVPLVMIPLTSDQPLVANRVQALGAGIALDKHNLSPAVLREALAEVLSNPLYKQQACLIGESLGQAGGYKRAADMIMNHF
AKV
>lcl|NZ_CP026520.1_prot_WP_042227387.1_3483 [locus_tag=PC41400_RS17645] [protein=MFS transporter] [protein_id=WP_042227387.1] [location=complement(3881681..3883138)] [gbkey=CDS]
MEVGEPKNSNQIGIFDTPYRALTIGIMLVLTTVAFEGLAITTIAPKLAQSLQGIHLYGWIFSAFLLSQLVGTLFMGQQVD
KRGVFTSMLVSFSIFVLGIVVAAVSFDMPMLIAGRVLQGFGAGALITCVYASVTLHYPDALRTQILAAFSMAFFLPSLIG
PYVAGLIASYISWRFVFWIVLPLLVLALSLTFRSFRDLQLRQALTGPAQTGSAQLADPKVIRAILLAIGTGLLLTGLGMI
TDWKGIVLTLGGLFVMIPLLRKLMPIGTFSVKKGLPATLVSRGLYVACYFATESFVILALTEVKGVSAEFAGLLVAAGSL
SWSAAAWLQAKLDARDHGRGRKRRVMLGIGIMIAGTVLVILALLLSGDGIMLFLLSQLITGFGVGLANPTTGTIALQHAE
PGNEGEVSATLQFVDSFCMGVSIGVSGAVIALAETLQWGISAGVLIVLTLQLVFVLLSFLASIRITELGHQEHHPTGHAQ
DKLPM
>lcl|NZ_CP026520.1_prot_WP_042226500.1_3484 [locus_tag=PC41400_RS17650] [protein=DMT family transporter] [protein_id=WP_042226500.1] [location=complement(3883521..3884420)] [gbkey=CDS]
MKTWHYACIVFLGGCCYGLLSTFVKLAYSAGFSVSAVSGGQYLFGALLSWLLVMVTKKRKLPFKRIVQLLASGIPFGLTG
VFYYNSLLTLNASLAIIFLFQFVWIGTLLEWILYKKKPSAVKIISIAVLLIGSILAANLVSKSGIVLSWKGMAWASLSAL
TFTAFIFLSGSVGKDVAPVQKSALLSTGGLIVVFVLLPPTFLLDMSVLVGIVPYGLLLGFFGVVLPPLLFSIGMPRIGPG
LGTILTASELPVAVSMSSLILSEHVNFLQWIGVVLILGGIAAGNVTFSKKAAVQQSSAT
>lcl|NZ_CP026520.1_prot_WP_042226502.1_3485 [locus_tag=PC41400_RS17655] [protein=LysR family transcriptional regulator] [protein_id=WP_042226502.1] [location=complement(3884623..3885510)] [gbkey=CDS]
MEIRQLKTFWTLASTRSFSRTAESLNYVPSTVTMQIKALEEELGVKLLDRLGKTVVLTDAGYQFLPYANKILNDIEEARH
VSSQSGELTGTVIIGADEVLCTYRLPALLRLFRERYPRVRLLFRPLSSHNLKQSLREGQADVVFMLDEPAVSVDLHTELL
LDEPFHMVVSPDHPLASRSTLSIEDFHGEHVLLSEKGCSYRNYFYRTLLKKGADSLTELEFNSAEAIKQCAMTGLGIALL
PEMVIKGEVERGELIVLPWDLSEIRFATQMLWHQEKWISPSFRGFIEMARSVLSH
>lcl|NZ_CP026520.1_prot_WP_042226504.1_3486 [locus_tag=PC41400_RS17660] [protein=alpha/beta fold hydrolase] [protein_id=WP_042226504.1] [location=3885690..3886697] [gbkey=CDS]
MRDYEIYPLGDVELQSGRTLPQAFLAYKTYGTLNAAKSNAIIYPTWFAGLHNDNEWLIGPGKALDPDNYFIIVPNMLGNG
LSSSPSNTPAPFDRGNFPLVTIYDNVRLQHRLVTEKFGITKLALVVGWSLGALQTFQWGTSYPEMVERIAPFGGTAKTRA
HAQVVFGSLIAALQADSNWKNGFYTRQPAAGLAAMGRAYAPWGFSQAYYLEQLYQAEGYPTLKSYLENYWDKVFFSFDAN
DLIAMLRTGIHGDISATPADNGNFEQALNRITAPALVMPGSTDLFFPPGDNAYEVEHMPDAVFRPVESKWGHCFGIGANE
PDSFVIDTHLKQFLQ
>lcl|NZ_CP026520.1_prot_WP_042226506.1_3487 [locus_tag=PC41400_RS17665] [protein=SDR family oxidoreductase] [protein_id=WP_042226506.1] [location=complement(3886837..3887598)] [gbkey=CDS]
MKMTGNTILITGGGSGIGLAFAERFIKAGNQVIICGRRENVLQKAKEQFPGILTRTCDLTVESERIALFDWVTVNYPEVN
VLVNNAGIIQHFNVLKADAKYNWSRFSEEITANLEAPIHLSMLFAPFFAEKREAAMINMTSGVAFTPLAVTPIYSATKAA
LHSFTMSLRHQLSATSIEVIEVVPPAVDTGLGGTWLRTHVEPLDAFADGIFKGLEEGRAEIGYSTSADRLRMTRDQVDEY
AEKMYNAMRASIE
>lcl|NZ_CP026520.1_prot_WP_042226508.1_3488 [locus_tag=PC41400_RS17670] [protein=MarR family transcriptional regulator] [protein_id=WP_042226508.1] [location=complement(3887666..3888112)] [gbkey=CDS]
MEKPSIAIPPSTRAQVGEVLLQISDDILDAINAALNEFHISESKFALLFMLFTTSRDQPLQPSEIADRLGIRRASVTKQL
IWLEEHHLITRTVHPEDQRMVRVTITEEGYRLLDLALPQYWQACANLTNKLTDEETVCLLHLLRKIDL
>lcl|NZ_CP026520.1_prot_WP_042226510.1_3489 [locus_tag=PC41400_RS17675] [protein=ABC transporter substrate-binding protein] [protein_id=WP_042226510.1] [location=complement(3888330..3889316)] [gbkey=CDS]
MKALIRLFLSVALLAGILAGCARTPEPNHNQAGAGAPEKAAEAKKEGGQWPRKYLDVLGREIVLEKKPEKVALLFFRNFE
HVFLLDEAPAAATDLNVLDEWESLAPYKSSRITDIGSMTNPNLEKLLEVGPDLIIMVSARYGSYGDQLEKIAPVITVDSN
ENNWQGALREYGKIFGKEQKAETEITRIEALIIDSKEQLKKFSDKTFGVIMLGDKAFWGFTTQFVFNKESGLALKPPSQY
VNMSTKGEQISLEGLAAMNPDYMFVADIGGSSKKLKGYLNDLESNSAWKSLNAIKNGRMYALDSSIAAGGPLGIELGVKT
IMKNMLAQ
>lcl|NZ_CP026520.1_prot_WP_042226511.1_3490 [locus_tag=PC41400_RS17680] [protein=AraC family transcriptional regulator] [protein_id=WP_042226511.1] [location=complement(3889387..3890232)] [gbkey=CDS]
MDINKLAEHYARTSFRVEQVCRYVREPGKSCDDYTEPFPGFVFPLRGKADFDFSGKVYTLTSGNVVHGGARMPLNRSVLG
DTKWEYLLVLYRICAPEPSGFCLSDTHFQLNTGQSPRLLELLKLLWRSSREPGGISTFQTETLFRCILEELFVCARNQNS
DGEQALFEQVSKYIHEHYMDGLSVRALAEQNDINENRLFYVFKKHTGMGPGKYLMAYRLNRAKESLLANDAPIGEIAKSL
GYEDALYFSRIFKKQVGVSPAEFRGSFRNNPYGFQDGPIPI
>lcl|NZ_CP026520.1_prot_WP_042226513.1_3491 [locus_tag=PC41400_RS17685] [protein=fibronectin type III domain-containing protein] [protein_id=WP_042226513.1] [location=complement(3890654..3891070)] [gbkey=CDS]
MKKQLIILTALASVLASGQAALAAENVQGENYNQKTASLIEPAAVVEYQVSWYAVSGATNYKIKMVRVDKPLDPPIVPVT
DLGKETLFKAPYLKSNTQYRVWVAAYDSKGKLLAQKTEEKKTVGEIGGIIVWQFGALK
>lcl|NZ_CP026520.1_prot_WP_174446775.1_3492 [locus_tag=PC41400_RS17690] [protein=glycosyltransferase] [protein_id=WP_174446775.1] [location=3891358..3892011] [gbkey=CDS]
MKNKWLVAILSISVLVFSMTPAVVSAQAKAVEKKDSACLSPKMVQLKVDMQKVWIDHTIWTRSYIVSAISNRPDQKDVLD
RLLRNQQDIGNVIKPYYGEAAGNKLADLLREHILIAGKIVAAAKAGNQADVKKLEADWHRNADDIAKFLSTANPNWPFKT
LQDMLYTHLQLITEIVLACLKGDWKADIAATDKNEIHMIHMADFLTDGIVKQFPDKF
>lcl|NZ_CP026520.1_prot_WP_042226516.1_3493 [locus_tag=PC41400_RS17695] [protein=hypothetical protein] [protein_id=WP_042226516.1] [location=3892343..3892582] [gbkey=CDS]
MNSVDKIRFMLLYGEAPEANCYGYMELNQYGDMIALYNWYGEEIDMYGVHEFVRVRSLGKFDDEDRDNFYSLLESDGVG
>lcl|NZ_CP026520.1_prot_WP_042226517.1_3494 [locus_tag=PC41400_RS17700] [protein=hypothetical protein] [protein_id=WP_042226517.1] [location=complement(3892740..3893423)] [gbkey=CDS]
MGALTVHLKATYLQMRIYIWCVILLLLLGKTAEFIVGLFIDNSQSRGLSGGNILILILPLIAIVLPLIYYKRIVHLGASR
EQYFKGLLFVFAVYAAVIALINCFWSALESNGVGNYPIPVNMITAFHWNDFGLAGCFLYQTAFYLMAMALLSMLISGYYH
PVGWLLWALLIAAIPVGTAIPSLRVHVVSFFKTLLFNGSLLAGIGFNLLLCLVFVAGGWLFTRGRTH
>lcl|NZ_CP026520.1_prot_WP_042226518.1_3495 [locus_tag=PC41400_RS17705] [protein=ABC transporter ATP-binding protein] [protein_id=WP_042226518.1] [location=complement(3893425..3894294)] [gbkey=CDS]
MSIALTCGNLTKKYGRTYALCNLDLQLEENVIYGLLGRNGAGKTTLLNTIAGGISPDSGSIEVRGKKLGKGELPNDFCFV
RENTRHFRGGRIIEVLKFAASFHPNWDWTFAHELLKAFRLDPDKKIRQLSRGTESLVGNIIGLASRAPLTLFDEPVLGLD
VLMREKFYKALLEDCANHPRTILLSTHLIDEIAPVAEKVYMIEAGSILLHDDMDEIRMAAHLIRGNSEAVASFTAGKRVL
HKETYGHGTLAAIYEKLNDGDRRLAREMDISIESLTLQSFFSYLIQGGH
>lcl|NZ_CP026520.1_prot_WP_241688685.1_3496 [locus_tag=PC41400_RS17710] [protein=GntR family transcriptional regulator] [protein_id=WP_241688685.1] [location=complement(3894301..3894669)] [gbkey=CDS]
MSVTFDDKQPIFQQVADIIEDDILNGTFRVDEQILSVAQFSQLFQINPATVVKGIGLLVNEGILYKKRGLGMYVAADAKQ
KIQIKRRERFHKELLSDLLDEADKLGLTTEDIMNMIKQLREE
>lcl|NZ_CP026520.1_prot_WP_042226522.1_3497 [locus_tag=PC41400_RS17715] [protein=ABC transporter ATP-binding protein] [protein_id=WP_042226522.1] [location=3894921..3895625] [gbkey=CDS]
MNKLIEFKNVAKEYKIGEVQIKALDGVDFSITEGEFVVILGASGAGKSTILNILGGMDTATSGQVFVGDQEITSLGEKKL
TQYRGEKVGFVFQFYNLIPNLNALENVEFATEVCKNYLDAKQILYKVGLQDRIKNFPSQLSGGEQQRVAIARAVAKNPLL
LLCDEPTGALDYVTGKSVLKLLQELNKDTKTCVVLVTHNSAIAPMADKIIKVKSGRIESVTLNKEKQNVEGIEW
>lcl|NZ_CP026520.1_prot_WP_042227389.1_3498 [locus_tag=PC41400_RS17720] [protein=FtsX-like permease family protein] [protein_id=WP_042227389.1] [location=3895625..3897952] [gbkey=CDS]
MKLFSKLIRDIRQSSGQFIALVLVVAVGAFFYGGLVSYSDTLRAYTKDYFKSHNLSDLYVYYNQISKEEAAGLSEIEGIH
KLEGRYTLEAKQVFENVKTSLTVHSIPANNEINTLTIVEGSIPSKKGEIVLDSHYAAEHQYREGDTIRIYTDGKEFTFTI
SGLCENVEYAKKNRTQDHKAQGFAYISEETIPDILGGGFTYNEIMIDAQEGYDVDQLGKSIEAQSSPLAYLNQVNKERTF
NYSQITQTIYNNTMMSKVIPLVLFLIEAIIVLLIMSRLIDSQRNQVGIMKALGVKSRSILLHYMGYPVLVSIVGSILGCA
IAALVFIPLVRGSNARAYSLPDITFSLSLFTLIPPIIFSGAFGSLACYLSGRSVLKERAAQAMRPKPPKKMKKLIIEKIP
GIWNRIPYSYKLIFRNLFLNKQKALASSVGIVVSTVLLITAFGTQTALQKVAGQTEDVHTYDLRVDYSVGPIPDPAELPS
GIKNRYGLATFPVEFIKDDGHENATLVVTEKDNSLLHFFDDQDNKISLEEGGVLVPKSYADKYHIAEGDRIKLKFADPEL
KNKSVDMKVLHISTQYSNPSFYITPAYLKSFDLDYSPSSLLIEAAGPADLASIRHFFEQDPQVDALTDKTDLKKSAQYIL
KQNSFVFIMFIICAVLLSFGAIYTITSINIYERNRELATLKVLGYQKSKINRLIFFENIILTTFAVLVALPISGYVYSLV
VKALSSTHQQIPNQLNLFVMSVAVIAAYLLTLLSNLLLRRKITRINMIESLKSVE
>lcl|NZ_CP026520.1_prot_WP_042226524.1_3499 [gene=ppsA] [locus_tag=PC41400_RS17725] [protein=phosphoenolpyruvate synthase] [protein_id=WP_042226524.1] [location=complement(3898119..3900713)] [gbkey=CDS]
MSSLVLGFQEMEKTQLSLVGGKGLNLGELSKLEGIQVPEGFCVTTAGYRKAIEQNETYQALLNRLTLLKADDRDQIGEIS
RKIRNIIRGAEIPSDVVTAVAHYLSRFGEEHAYAVRSSATAEDLPHASFAGQQDTYLNIIGVDAILQHISKCWASLFTDR
AVIYRMQNGFDHSQVYLSVIVQRMVFPQASGILFTADPITSNRKRLSIDAGFGLGEALVSGLVSADCYQVQEGEVVNKRI
ATKKLAIYGRKEGGTETVQLNPDQQMAQTLTEQQILQLARIGRQIEAYFGCPQDIEWCLAHDTFYMVQSRPITTLYPIPE
ANDQENHVYLSVGHQQMMTDPIKPLGLSFYLLITPARMRKAGGRLFVDVAPMLAAPVGRETFLNNLSSDPLIIGAIKTII
ERDFIKLLPNDQTAPSPGRNNTDMRAPIENDPTIVSELIKRSQASIEELKQNIQTKSGSDLCDFILDDIQQLKKILFDPQ
STAVFMAAINASSWINENMKEWLGEKNAADTLSQSVPGNITSEMGLALMDVADAIRPYPEVIDYLQHAKEDHFLEELVKF
DGGREARNAIYDYLSKYGMRCTGEIDITRTRWSEKPITLVPLILGNIKNFEPNAGKRKFEQGRREALEKEQELLDQLKQL
PDGEQKAKETKQRIDLIRNFIGYREYPKYGYISRYFVYKQALLKEAERLVQAGVIRNKEDIYYLTFEELHEVVRTKKLDD
SIIGKRKDEYTFYEKLTPPRVITSDGEIITGEYKRENLPAEAILGLPVSSGVIEGRARVILNMEDANLEDGDILVTSFTD
PGWTPLFVSIKGLVTEVGGLMTHGAVIAREYGLPAVVGVENATKLIKDGQRIRVHGTEGYIEIL
>lcl|NZ_CP026520.1_prot_WP_042226525.1_3500 [locus_tag=PC41400_RS17730] [protein=SMI1/KNR4 family protein] [protein_id=WP_042226525.1] [location=complement(3901043..3901501)] [gbkey=CDS]
MYERLADKLKTTTALKWFPGHGAEESWIAEAEEELGFSLPPSYRWWLIHYGNASLNGGNILTLVSPEHREYVDSDLLYIH
RLNKEEEWLVSRFPHRLDLFVPDDDELYYFDTSSKNEQGEFPIMRYDLMNDLIETYALTFAEFLEQLIDERS
>lcl|NZ_CP026520.1_prot_WP_042226526.1_3501 [locus_tag=PC41400_RS17735] [protein=DUF11 domain-containing protein] [protein_id=WP_042226526.1] [location=3901822..3913662] [gbkey=CDS]
MAFLQRYAANQTGAITFTGNTLGLSRSETAGVPGTVDSIGAYITTDTTLQYGTYPAGTTSLYTSNSSAAVLTLPAGSTVL
YAELIWGGSYLDGTVNNTAFINNPITFTTPAGTSSVTQDAATVYNVTFAVSPSSVLAYVRTANVTALVSAGGTGTYTAKS
IVGTLAVPDPTANHAGWTLAVVYQNPSLPFRNLSLGVGAEIVQSTSGAVNVVISGFATPAQGTLKGRALISAQEGDANKT
GDQALFGPTASSLTALSGANNFAANFFASQINKDDGTLDTSGTFGTRNAVNGAPGTLISGGRQGWDITNVDISSTLKNAQ
TSAILQLTTSGDGYIVNANAIQIDINAPLVSVTKTASVTGAVIGDTVTFTIAVKNTGLVDASSTVLTDPIPAGSALVANS
VTVNGTARTGTDPSAGILLGTISAGSMTTVTFRVTVTSLPSPAQLSNQATASFSFQSVAGGPVISGSVPSNTLAVPVYQP
VIGLVKTANTSTATVGDTVTYTLTAANTGSIRAAATLTDPIPAGSTFVTGSVTVNGTSLPAANPATGFTIGNIDAGGGVV
TVTFQTVVASVPTGGRLIDQGTASYTYSPPDGRTLSGSAVSNTLSIPVSAPNVTVVKTAGSTDAVVGDTVTYTITATNNG
IVPVTGVVVTDPIPAGSSFVTGSVTVNGTAVPGANPATGVPVASIAAGSSAVVTFRVNVTSVPSPAQLSNQATVSFSSGA
FTGSSLSNTITLPVYQAVVSAVKSVSPASATVGATLTYTVTVKNSGNLSVSLNVSDPIPAGSTFLAGSVTVNGTPLPAAN
PAAGFAAGSLAAGASSVITFQTVVTSLPATSRLVDQATASYTYQPPDGRTLSGSAVSNEVSVPVSAPNVAAVKSGSKTAA
TVGDVITYTVVAANGSSVPVTGVVITDPIPAGTTFNAGSVTVNGTAVPAANPSAGVPVGTIAAGASTTLQFTVAVTSVPS
PATLGNQATVSFTSGAFTGSSVSNTVSVPVYQPVLSAVKSADTARATVGDSVTYTVSLTNSGNYNASITLTDPIPAGSSF
APNSVTVNGLPLAGADPVTGISVGTVSPGTTSTVTFQTVVNSLPAPPQLSDQAVAAYTYRLPDGRTLTGTVSSNMLTIPV
SAPNVSVVKTTSATDAIIGDTVTYTIAVTNNGIEAVSGVVVTDPIPAGSSFVSGSVNVRGAPLPTANPVAGIPVGSIAAG
ASAAVTFAVQVNTLPSPPQLSNQASASFASGAFSGTAVSDLLIVPVYQPVLAAVKSAGTSRVTVGDTVAYTVTITNSGNL
SAQLTVTDPIPTGSSFVANSVTVGGVPVPGANPAAGIAAGVAGPGAAVTVTFQTIVNTLPSPPVLTDQAQVSYTYRPPDG
RSLTGSVSSNSVTLPVSLPNVTIAKSPSAPDAIPGDVITYTLTVSNSGTGAVSNVVVTDPIPAGSVFVAGSVKVNGTAVP
AVNPANGISAGTIAAGASSTIVFSVQVTSLPSPPQLVNQASVSYTSGSFSGSAASNTVSLPVYQPIVIVKKSANLSQGTV
GDLVTYSITASNSGNIGVTASITDAIPAGSSLVSNSVALNQVPLPGANPAGGIAAGTLAPGASTTLTFQVVITSVPTPPQ
LVDQAAVSYTYQPPDGRTVSGSASSNTVVIPVSAPNVVISKSASAADAVVGDTIAYTLVAVNNGIENVTNVIVTDSIPDG
TTLVPGSVTVNSVPVPGANPSVGVPAGLIVPGGAAAVTFSVLVNSLPASGQLVNRGSVSFTSGTLSFSSLSNTVTVPVYQ
SILSIAKSANTVNATVGDTITYTLVLSNSGNIAASTTVTDLIPAGSVLLDNSVTVNGSPRAGADPSTGIPVGSVAPGVPV
TVTFQTVVNSVPASSILLDQASASFVFVPPDGRSVTGSAVSNTLVIPVSAPNVTVTLTTASTDAVLGDIVSYTITATNNG
IENVTGVLATDSIPAGSVFVAGSVVLNGTSLPTANPSVGVPVGIIAPGASAVLSFQTSVTSLPSPPVLSNKAIVSFTSGT
FTGSSTSGTVLVPVYQPILAAVKTANFSNATVGDTIAYTIAVTNSGNIAASLVVTDPIPPGSTFLANSVLVNGLPLPGAD
PTTGITAGSLLPGATTLVSFETVVVSVPPSSRLNDQAAVAFTFQPPDGRTISESVSSNPVTIPVSAPNVAVTKTAGSTDA
VVGDTIAYTITATNSGIENVANVVISDPIPAGSVLVPGSVTLNGLPVPAGNPAAGIPAGIIAPGASAFLRFRVTVTSLPS
PAQLLDQATVSFTSGAFTGSSVSAPLSIPVYQPLITLAKSGNSTNATVGDTITYSILVTNSGNIDAAATITDPIPAGSAF
FVNSVAVNGTSYPGSDPSTGIAVGSIAPGSSTLVTFQVIVTSVPAPPSLVNQASASFTFVPPDGRIIQESAVSNVVSIPV
SAPDILVTKTAGSTDAAVGDTITYTITASNAGISGVTNVIVTDPIPTGSALVPGSVTVNGIVKPGADPASGIAVGSVAPG
AVATVSFQVTVTSLPSPPVLSNQARVSYTSGAFSGSSLSDPVVIGVYQPILSVTKSANTANATVGDTIVYTILAANSGDI
AAVTTISDPIPAGSAFIANSVVVNGQPIPGADPTVGIAVGPIPPAGTAGVTFQVVVTSVPSPPTLVDQATAAFTFQPPDG
RTLSGTALSNAVSIPVSAPNVTVGKSASAADAVFGDTVTYTITAANNGIEAVTGVIVTDPIPAGSVFVPGSVTLGGVPLP
LARPDAGINAGTIAAGATATVTFQVTVTSLPSPPVLSNQAAVSFTSGVFSGTSYSDPVTVPLYQPILQLAKSANFSNVTV
GDTLTYTIQTTNSGNIAAALLLSDPIPAGSQFIENSVSVNGIPVPGASPVAGINAGSVLPGSTASVMFQVLVTSVPNPQQ
LTDQATAAFTFLPPDGRIVNGSAVSNTLSIPVSAPNVSVVKGTTAADAIFGDIVTYSIAVTNNGIEAVTDTVVTDPIPPG
SAFLPGSVTVGGTARPGTSPVTGIPLGTLASGSTAVVTFQVTVQSLPTPPQLYNQASVAFTSGAFTFSSVSNPVVVPIFQ
PILTVAKSGSSDNATVGDTVTYTIVLTNTGNLAASASVTDPIPPGSSFVPNSVTLNGIPLPGTDPVTGIPTGSIAPGGSA
AVTFQVLVTAVPVPPLLIDQASAQFTFVSPDGRTNTGSGASNTVIVSVSNIAVTAVKETDLRSSILGDTIRYTITVTNKD
ADLIREVVLTDPLPQGSAFVKGSVTVDGVALPDADPNTGIAIGQLASGASVAASFQASVVSLPSPPELTNQAVVTFVSGA
EAGSIVSNSVTVPVLQPILTVPKTGDTDSVAAGDTVLYTITVANTGSTGAFTTITDALPPETILVPDSVTVNGTVIPGAD
PAAGITVGTIAPGASASVSFRVRVTVIPPEGVLVNRALSAFSFQTPDGRTHEGSSASEPVVIPVTPHSVSLFKNSTVEAA
AIGDVLAFGITVTNGGQAPAADVIVTDLLPEGTSFVPGSVTVDGTALPDAQLQAGIALGSVPPQSTVLIGFQLEITSIPE
AAQLVNTASASFTYADHSYTVSASSDAIFVYKYGIEAVKSSDQTIATIGSEIAYNVVVTNTGTIDASSIVLTDRLPAGTQ
LDAGSIRIDGQLQPGANLSAGLSLPSLGPGGSASVSYSVTVTEQPDSGQIENQARIESSYVLPDGKTEWEVVSSNLIGIS
VHSPVIIAEKSASPGIRVLGDVVEYTVAIRNSGTLAADVTLQDLLPPGSAYVPDSVTVNGEPRPAADPSDLAIGLLAPGS
AVTAAFRLTITSLPAEGLLRNRATAKFTYRLPDETDIYTGAAASNEVVLPVFAPAVLIHLNVDSHEAASGDILTYTAVIT
NTSRAPIVEIFTENDLSAGLAFVSGSVRINGMSAPNADITAGVLLGELQDGESTTVQFAVAILPATYGLVTLRQTAGFSV
RLPDNRIISGSQQSNPVTIRIVQHEE
>lcl|NZ_CP026520.1_prot_PC41400_RS17740_3502 [locus_tag=PC41400_RS17740] [protein=transposase] [pseudo=true] [partial=5'] [location=complement(3914150..>3914944)] [gbkey=CDS]
LDRYFPEFLTVFKDWECKSAIPNAKPSSTAT*ACFCARRDVAAAIKRSCKARAWSRSNTESEGSSRLFCWHQARF*VG*N
GAKVVADTVRMASK*A*GIGSKVR*TTSADPSCKASIGRDTISDFLAEVGDIYQYRHPKQITKLAGLNLKTNSSGTHIGQ
TRITKRGRKRLRALLFRVMMPLVAKNSAFRTLHDYYTKRPVNPLKKMQSLIALCNKLIRILFGILQKGHEFSEEKMIPDI
PRFGRYHKQLNDRFFYVDDQSRAR
>lcl|NZ_CP026520.1_prot_WP_042226528.1_3503 [locus_tag=PC41400_RS17745] [protein=GNAT family N-acetyltransferase] [protein_id=WP_042226528.1] [location=3915191..3915610] [gbkey=CDS]
MNIRVANELDYPALRKIYLESRRKSFHWADIEAMTLDDFDKHTEGEHIFLVEENNQILGFASLYLPDNFIHNLFVHPDFS
GKGAGGLLLQASIEKMEKPIRLKCVSKNHAATKFYESKGWKKVVEEGTPHEKYWVMEYN
>lcl|NZ_CP026520.1_prot_WP_042227391.1_3504 [locus_tag=PC41400_RS17750] [protein=DJ-1/PfpI family protein] [protein_id=WP_042227391.1] [location=complement(3915842..3916441)] [gbkey=CDS]
MKIQIVLFDGFGELVSLAPFEVLKRAIEEGAPFTVDLISSEPKQEIAASFGVTVKLNDFLRMDNRPDLLIVPGGGWNHKA
EHGARKQAELGTLPAMIREMHNEGTIVAGVCTGGMLLAASGILDGRKATMHYLAHDEISEYGADLLRYRIVDQGNIITAR
GVTSGLDLGLWITERFAGAKIAAAVEYRMEYERRGVVWS
>lcl|NZ_CP026520.1_prot_WP_241688686.1_3505 [locus_tag=PC41400_RS17760] [protein=sulfatase-like hydrolase/transferase] [protein_id=WP_241688686.1] [location=3916906..3918693] [gbkey=CDS]
MIPISRHLKRPNFLVFLVDEERYAPVYENEEIKEWRERNLVAQNLLKSHGLEFHRHYIGSAACCPSRATLMTGHYPSLHG
VTQTTGIAKEAFDSDMFWLDRNTVPTMGDYFRAAGYRTYYKGKWHISHEDITVPGTHTSLPSYDPLTGVPDKSLEALYSN
ADRLNDFGFSTWIGPEPHGRNPRNSASSAGIGLSGRDEIYAAEAVELIEALDRLNRQDNLAEPWLLVASFVNPHDIVLYG
AITERLPQFKFEVEPMPEVAPSPTLNESLFTKPRCQASYRNLYPKALQPIIDQPFYRKLYYQLQKNADRQMLKVFDALTR
TSFYDETIVIFTSDHGEMLGAHGRLHQKFYCAYEEILHVPFLIHNKHLFPQHASLHTLTSHVDLLPTMLGLANVNIEEIQ
NSLRNTFSEVHRFAGRDLSPCILGKGPSTIDEQPVFFMTDDDVTRGQHQISPLGVPYSSVIQPTHIETVIAPLYRNGRKE
LWKFSRYFDDVQFWSQPGIQDVTLHPVGGAASGTPDSQWVTQVKTQPVPDEYELYNITLDPLETCNLAHPAYATEQTRTI
QLRMMHLLADQRQQKRLTPVKKHFLAPHTTAQTNF
>lcl|NZ_CP026520.1_prot_WP_042226529.1_3506 [locus_tag=PC41400_RS17765] [protein=sialidase family protein] [protein_id=WP_042226529.1] [location=complement(3918769..3919986)] [gbkey=CDS]
MANIQISPSTSMQFEPSIAVNWLNPSVMVVVAVDFSTGSPNIGLYKSIDAGATWSTTLLSLPTGFVGIESPHIDYLFPNS
FVVLAHAFDENGLSGTVISYTSNDNASSFGPPVVVNQGYGQFVNNDQVVVTTDKAGSSPFLGSIYTGYTHDYNTEFTPGN
TIFFNRSKDGGNTYSDPTLLSSVTEFEEFPGIAITLNGILIVGWITQPPGVSDFNIRTSQDGGTTFDGETMVTSVVVVPS
PLPGYQFRCLTYPSLSGDISNVPSTRGNVYAVWQDYRQGYSDIYLSISTNFGVNWGAPMPIADSPVGSQNFFPAITVSPS
DGAVFVSYYTNRVNPPDLDVFLATSRDGGVTFTNTRLTTTSFSVAGLLVIGDYIGNAIVPQTDRLVTVWTDTRTGIANIW
FEDNQ
>lcl|NZ_CP026520.1_prot_WP_042226530.1_3507 [locus_tag=PC41400_RS17770] [protein=hypothetical protein] [protein_id=WP_042226530.1] [location=complement(3920032..3920397)] [gbkey=CDS]
MPLTTGIITNTGINPATNLVINIDNDNLSFTSNVVYRIYVWNNLVSKALVYSNSLFINANTSQILNFSITGNSAYQVQFL
VTGTVPTDTVITVFGTDSSGNVIQHQRILQQELTQIGQLSP
>lcl|NZ_CP026520.1_prot_WP_042226531.1_3508 [locus_tag=PC41400_RS17775] [protein=aminotransferase class I/II-fold pyridoxal phosphate-dependent enzyme] [protein_id=WP_042226531.1] [location=complement(3920801..3921985)] [gbkey=CDS]
MFSEEGTGRFLSSRVRDIRPSGIRAFFDLNAAGGDTIALGVGEPDFVTPEKVREACIQALREGQTQYTSNAGSMELREEL
SAYLADSFTLSYDPEQEIIVTVGSSEAVDLALRAVINPGDEVLIPAPSYVAYEPITHLHGGKIVEVAAAVQEQFKLTPQA
LQAAITPHSKALMINYPCNPTGTVMTDQDWLPIVELIIKHNLVVISDEVYAELTYGRKHVSIASFPGMKERTIVISGFSK
AFAMTGWRVGYACGPRELIAGMLKIHQYTAMCAPTLAQIAAIESLRHGLAAKDEMVASYNERRKLFVAGLNAIGLTCHEP
EGAFYAFPSIASTGMSSQQFALRLLEEAKVAVVPGHVFGSGGEGFIRCSYAASLFDLEKGLERMGRFMRATQPI
>lcl|NZ_CP026520.1_prot_WP_042227394.1_3509 [locus_tag=PC41400_RS17780] [protein=D-alanine--D-alanine ligase] [protein_id=WP_042227394.1] [location=complement(3921978..3922916)] [gbkey=CDS]
MKIGVIMGGISSEREVSLQTGQEMIRHLDRNRYEVVPIVIDRRADLIIQTQQAGIDLALLALHGQYGEDGTVQGALETLG
IPYTGSGVLASSLCMNKQLSKRLLKTVGVHTPEGLCWQGTDDYEPQAVKQLGYPVIVKPNMGGSSIGIQLVHNEKELLTA
VRKAGSFDQAILIEPYMKGTELTCSIVDGEVLPIIGIRSAHSEWFDYKAKYEPGGAEEKVIQLPPATRQRVREAALASYR
LLQCKVYARVDMILCEDVPYVLEVNTLPGMTENSLLPKSAAASGMTFTQLLDRIIASSILERKQEWGMVQDV
>lcl|NZ_CP026520.1_prot_WP_042226533.1_3510 [locus_tag=PC41400_RS17785] [protein=PLP-dependent aminotransferase family protein] [protein_id=WP_042226533.1] [location=complement(3922944..3924434)] [gbkey=CDS]
MFQDFKLVGDRPVALQVKEYIKRLIIKGVLQADQKLPSTREMSSLLKVSRNTVIAAYEGLEDDGFRYAIPGKGSYVAATE
GHLTTDAGGADGSAAWQIDWKARMNEYAVSAVELDMMKQGIRANKGVISFTSIAPDEGLFDLGDVRRAFMDRMAIEGQVL
LNYGYAKGYQPLIDYLIRYMENKGIDTSGKDMLITSGFTEGFDIVLSALRPSARRGAAICENPTHHTAIKNLKLQGFEIT
GIPMERDGIDVNRLEKVLQKQTNSFDLAYLTPSYHNPTGIVMSPAKRSAVIKLMTRYQIPVIEDGFNEELRYSGAHVAPL
IATAGQGNGVIYIGSLSKVLFPGLRVGWVLGDRALINHLESIKRARAIHTSTIDQSILYQYLLNGNFDKYIKKARMEYKR
KYELTKACCEAHLPEAQLSGDGGLHLFLTFPASVDTRRLLEACTEQGVIFTPGDKFFIQEGEGTNTLRLGFSRVTDENIE
RGIRIIGQEARSFLQA
>lcl|NZ_CP026520.1_prot_WP_042226535.1_3511 [locus_tag=PC41400_RS17790] [protein=PLP-dependent aminotransferase family protein] [protein_id=WP_042226535.1] [location=3924742..3926193] [gbkey=CDS]
MLKVNRDDERPIWQQLLDQAIHNITTGKWPPGELLLPSRELAPLIGVSRSTIQIVYEELFSRGYTVTSRRGGTRVSKWTY
ASRPTEVAAPQGPVLPELPVLNSAIGHLHSWFGDKENRKAEIDFSPHEPYLDEHFQKNWRQSFLHASTKTDLDSWAYGDS
YGFLPLREQIQRYLSLERGVHVNTDQILLTSGAQHSIDLIAQALLHEGETVSVEDPGFPAAWMAMKYRRMQVVPVPVDEY
GLCVNRIHSQSKLVFVTPSHQCAVGVIMSEPRRQQLLHMAAGQRFWIVEDDYDSEFRYRGDPLPTLFSQRPQNTLYMMSF
SKMVAPGIRISAIIGPKEAIRQLARVHELTYRHLPIMEQLTLTHFIEHGHFMRHMRRVRNIYRRRHEAMTKAIITTGLSE
HFKLSGVETGLHMLLEANKSFDEEAVTNLALEKGIRVYPLSTYCLESDRKGWVLGFAKVDEEAIEEGISRLAELLLSYKH
LHP
>lcl|NZ_CP026520.1_prot_WP_042226536.1_3512 [locus_tag=PC41400_RS17795] [protein=YggS family pyridoxal phosphate-dependent enzyme] [protein_id=WP_042226536.1] [location=complement(3926190..3926933)] [gbkey=CDS]
MGHPVEENLKIVRQQMELACQASGRKIEDVKLLLATKTVPLDKLQMAIQAGETLFGENRAQELRDKFPLMQQYNQVEWHF
IGHLQTNKVKDVVKYVTFIHSVDRLKLGQALHHQLVKENKTMDILVQVNTSYEESKFGVPPETALELVEQLSQFETLNVK
GLMTIGKLNAANDETRHCFRLLKLIQSQIREKKIPRVEMDILSMGMSCDFRVAIEEGATMIRVGTSVFGQRYLPDSYYWN
DNAHRDD
>lcl|NZ_CP026520.1_prot_WP_042226538.1_3513 [locus_tag=PC41400_RS17800] [protein=pyridoxamine 5'-phosphate oxidase family protein] [protein_id=WP_042226538.1] [location=complement(3926984..3927520)] [gbkey=CDS]
MDRVRYKIREVLDKRKIDTFLQQARIGQLGMVDGHLPYVVPLNFVWTNGKLYFHGATGGRRNQVMDANPEVCFTVCEEYG
TITDPVPAKTDTAYMSVMIFGKAQPIVDLDEATHMLQEMMHKYVPGYYNRPLPRQHVDKYRSAVFGGPVQVYRIDPYHIT
AKENAVEEEKMFKPGKTV
>lcl|NZ_CP026520.1_prot_WP_174446784.1_3514 [locus_tag=PC41400_RS17805] [protein=erythromycin esterase family protein] [protein_id=WP_174446784.1] [location=complement(3927766..3929088)] [gbkey=CDS]
MVAMVSAALLVGGCAKEGETKTVVVTESAEPVQNDVISSIKSQAKPLKTVDPTQPMDDLQPLKSMIGDARYVGLGENTHG
SSEIFTMKFRLVKYLVTELGFTNFGMEEDWGSGLKLNEYIQTGKGNPREFLNLLYPTDEIVVMIDWMKEYNADPSNKRKI
QFIGLDIKSLDQGVYQQVTDYVKRHYPDMLAEVELNYKELRLSTGSLQDYMKLSPEVRERFKGNAEKVVRLLEEKTKQAD
KKAETTELIWAKGTATAIRNFTSMLIPDDYPSMIKLHEQYMADHAVWAGEALDGKTMVWGHNIHVAKGIIDKELYPDGSG
NFLKKRLGDQYIVMGSTTSEGDFTLFSEYRPDGNSKLSTEKIPQDVNSSNYMLGKVSQQMFLLDARHLEGPAKQWVGEKR
PLLSLGGQLLPNAPVYFDASLREQFDILFHIRKTSPSHMK
>lcl|NZ_CP026520.1_prot_WP_042226540.1_3515 [locus_tag=PC41400_RS17810] [protein=CPBP family intramembrane metalloprotease] [protein_id=WP_042226540.1] [location=3929440..3930315] [gbkey=CDS]
MISNAATLSTEEQVHRARKGLRLFLSILLVFSIIMNTAVILTKSIPLIVVYMFTPALSSILARLLLREGFKDVSFNLGSA
KIWKGIGLAVLIPIIICFITYSIAWFSGLAIFQLPVGGSLEPIFHMLGLQHLPTPLNFILIVVVTGILGSLINLIPVMGE
EVGWRGYMLTRLVNAKVPRPVLVSGLIWATWHVPIVIAGLYVEGPSMILTVAGLYLCIVPFGYIIAYLRLATGSVWPAVI
IHSAWNAVVQGPFARATTGNHTEIWVGESGMITALIILIAAILVSRRKLIK
>lcl|NZ_CP026520.1_prot_WP_042226542.1_3516 [locus_tag=PC41400_RS17815] [protein=SDR family oxidoreductase] [protein_id=WP_042226542.1] [location=complement(3930558..3931424)] [gbkey=CDS]
MIVIMGATGTIGSALLERLADLGVPAKALSREPEKLRDQIGEKGRSTIEIASADASDPESLRRAFTGASQLFLTMANSPR
QIELETSIIQIAAESGIEHIVKISSPAFEQRSPVAVAGWHQEIERTLRESGLTHTVLRPYAFMQNLMRLAPTITTQNVFF
GSMGDSPCNFIDCRDIADVAAEVLTNRKVSGQIYTLTGSETFSYPQIASQLSVLLNRPISYINLEPQVLLRNLIENGHMP
PWLANHVVEIQTMSTVVPESPNDTVKRLLGREPRTLEAFLHECVENFR
>lcl|NZ_CP026520.1_prot_WP_129112507.1_3517 [locus_tag=PC41400_RS17820] [protein=hypothetical protein] [protein_id=WP_129112507.1] [location=complement(3931421..3931930)] [gbkey=CDS]
MYKMKYTNQTDDVYGIKAAGSKKKTFVLWREVVMAYAAPAIMAGIGGLVTADRSLQIGALTTIGGASALMALMIGLLLRS
RVGHIRWVIGAPHLVVVGVFALAGSSFGLCAAWMTSGLLEFMIPSNHLAWVGRVWIDFPLSGFLASTIVTWRWRLAATTN
FSSKRRRKK
>lcl|NZ_CP026520.1_prot_WP_042226544.1_3518 [locus_tag=PC41400_RS17825] [protein=MarR family transcriptional regulator] [protein_id=WP_042226544.1] [location=complement(3932039..3932605)] [gbkey=CDS]
MNNNHKRINDMDAIPSLAQSKRQVDRYNLDIDAQAILVASRLMAAGAKLGHASEIHFSRFGLSTGRYRLLADLEDNEGEE
LPSQLAEHLGVTRATVTGLIDTLERDGLVSRRSSRDDGRQRSVILTEEGAKKLREMAPEHFARLEAMVGLLSIDERSVFL
DLLGRVTQGISALTDELDSKPKGNISKE
>lcl|NZ_CP026520.1_prot_WP_042226546.1_3519 [locus_tag=PC41400_RS17830] [protein=excinuclease ABC subunit UvrA] [protein_id=WP_042226546.1] [location=complement(3932815..3935070)] [gbkey=CDS]
MSESNQEYIVISAARENNLKNVSLRIPKRKITIFTGVSGSGKSSIVFDTIAAESTRLLNENFSMFVRTFLPRYPQPDADA
IENLSMAVIVDQKRLGGGSHSTMGTITDISPILRLLFSRVGQPYVGQANMFSFNDPQGMCPECNGIGRSLSIDMSKALDK
SKSLHEGAILLPGYSVDSMDWTIIVQSGPFDPDKKLGDYSEEELEQLLYGKARKVATLFGGKTINITVEGVIEKFTNKYI
KQDVKTKSERTQKTVAPYISEGPCSSCHGARLSQAALGCKIQGRNIAEMSSMEVGQLIRVIREIDDSVAAPIVKSLTERL
QHLVDIGLDYLTLDRETDTLSGGESQRVKMVKHLSGSLVDVTYIFDEPSIGLHPRDVHRLNELLQKLRDKGNTVIVVEHD
PDVIKVADHIVDVGPHAGSRGGTIVYEGSFEGLLEADTLTGTHMKRQLRMKQDCRQPSGKLSIKDASLHNLRNVSVDVPT
GVLTVVTGVAGSGKSTLINEEFLSRHPDAIVIDQSAVGVSTRSNPATYTGIMDDVRKAFAAANKVSQGLFSFNSKGACEN
CQGLGVVYTDLSFLDSVKLPCDVCGGRRFKEEVLAYKLNGKSIAEVLEMTVEQALDFFQLKEVVRKLQAMSDVGLNYITL
GQPLSTLSGGECQRIKLASELHKKGSIYVMDEPTTGLHMSDIGHLLEIMNRLVDAGNTVIVIEHNLDVISQADWIIDMGP
DGGSKGGQVVFEGTPPQIIQAEQSITGRYLM
>lcl|NZ_CP026520.1_prot_WP_042226547.1_3520 [locus_tag=PC41400_RS17835] [protein=SRPBCC family protein] [protein_id=WP_042226547.1] [location=complement(3935379..3935798)] [gbkey=CDS]
MSLTLSLDFQYKTSIEKLWSALTDSSKLAKWVADIHTGQAMENDFKPVVGHRFQFRAQPTQWWNGIIEGEVLVVDEPNRL
SYTWASGEKHTVTWTLEDLGDGKVNLHLEQTGISNEQALNGAKYGWTKWCGGLENVLEQ
>lcl|NZ_CP026520.1_prot_WP_042226548.1_3521 [locus_tag=PC41400_RS17840] [protein=metalloregulator ArsR/SmtB family transcription factor] [protein_id=WP_042226548.1] [location=complement(3935795..3936124)] [gbkey=CDS]
MSEDHQLRDVFDAIADPTRRRLIHLLAQAEEIPLHELTAQFQMGRTAVSKHLTILKEAGLVHDRKVGRETRFRLNASPLR
EIQDWVAFYSKFWSTNLLRLSQLLEEEEE
>lcl|NZ_CP026520.1_prot_WP_042226550.1_3522 [locus_tag=PC41400_RS17845] [protein=helix-turn-helix domain-containing protein] [protein_id=WP_042226550.1] [location=complement(3936304..3936645)] [gbkey=CDS]
MSMAEYKGKVKHIQDTPFGYTMSVISGKWKMVIIYLLAENQLVRFNDLKRQIGAITYKTLSSQLKELEADGLVKREEYPQ
IPPKVEYSLTDKAKTLLPILEGLCEWGVKNQPS
>lcl|NZ_CP026520.1_prot_WP_042226551.1_3523 [locus_tag=PC41400_RS17850] [protein=RidA family protein] [protein_id=WP_042226551.1] [location=3936751..3937143] [gbkey=CDS]
MTTAKTYNHGVAWEEACGLSQGYSVNGTIYISGQFSHDMQGAFVGEGDFEAQARQTLENLDRVLAGFGVTRSQIAEMEIF
LTNPQEHFEKCVVLYKEYVGDHRPAGTLVGVSGLAFPHQLIEIRAVAHTG
>lcl|NZ_CP026520.1_prot_WP_042226553.1_3524 [locus_tag=PC41400_RS17855] [protein=SMI1/KNR4 family protein] [protein_id=WP_042226553.1] [location=3937206..3937517] [gbkey=CDS]
MERTLQVTLPTAFRDYLSIMNGQHDEIPFIGYNCFLSIADIIETWSMMNELFEDEEQVDWVNEDRIKPVIWSNKWIPFTD
FEASSRLILDLDPGKNGITKTNL
>lcl|NZ_CP026520.1_prot_WP_053228630.1_3525 [locus_tag=PC41400_RS17860] [protein=hypothetical protein] [protein_id=WP_053228630.1] [location=complement(3937869..3938882)] [gbkey=CDS]
MIQINQDRRVNLDDIAAEHLAYYQSKIEKKIDTYLAGRLPREQRAFLQHVRSKLRTILIGSPEELERFSEETERQYPAVV
AQLRQAKKPKKGKPVPFPEKVRKLFNYEAFTKRYDSSKWGAYRLAMLLGVQVCVYCNRQYTHTYNSDIGRVRPHFDHFLD
KATHPYLAISMYNLIPCCSVCNSSLKGSETFSIRTHLHPYLEGTGEHIRFSVRFKGSSTDYTKLWVGNSDVFDIVLKADP
GTKAPLTFLKRAIRTTRTFKIKELYQFHKDYAAELVVKAQIYDDAHIRSLLADFPQLFRSREDVLRMLTANYTKAEDADK
RVLAKLTMDIAKEFGLS
>lcl|NZ_CP026520.1_prot_WP_053228631.1_3526 [locus_tag=PC41400_RS17865] [protein=AAA family ATPase] [protein_id=WP_053228631.1] [location=complement(3938879..3940714)] [gbkey=CDS]
MELLYIWVDHYNGLFERQGFNFGGPFRFHYNADTLALTVKPNNHHIPGFFSPESVDEDLAYISNVTAIVGQNGAGKSSLL
DLIKRQISSGRVLQDTIIAVKDTEGNYALYYPLGMELTVKRSGEVPEFKRKPYIIESSKIPAKIGNGSRTRIGSIPDLQA
MTSVYFSNIFDLRGEQTPKDVVNLSTNHLLSDVKENSQYLQTHEVAVYRCREIERQLHLLNMRYHLDVNLPFAEPETLAI
NFTYEDLDELPNSSSELSELFERMEYEVSAGTLQERLLLTFVNHIFVEICNMDLDHHVTEHIKIYNIAHGKTWSDRVISW
LETVRIKLGKGKGINAVKRIIEGTVKLYRYLDLEIRNGKFTLDNQGLRMPVSESDTFESFFGLYARSLATHEYALYDWRD
LSSGEQALLGMYARFNTLMKRDVIKHDHILIMIDEGELYFHPEWQKSLLLHLLQFFTGLFRSEERRRSVQIVITSNSPFI
VSDLPASNIIFLRKTGALSQVVDGLDDHKQTFASNIHTLLAHSFFMENGLIGEFAKSKINAVIELLVHGKTSEIKANKER
IEKTIHLIGEPVIKAKLISMLNDRIAVHFLNIEDQINDLQRQIDELRGASK
>lcl|NZ_CP026520.1_prot_WP_042226555.1_3527 [locus_tag=PC41400_RS17870] [protein=DNA mismatch repair protein MutS] [protein_id=WP_042226555.1] [location=complement(3940986..3942992)] [gbkey=CDS]
MNEKALQRLEYGKIKEALAEYAVSYLGKRQITELLPLSAMSVVRAKLDETAEAKFLLQHGASVPIPSLEGMENILDLLGT
GYVFSEKDFMHIHQFLTSCAQLMRYMASKSGAAPQISSYAASMYTMDALKSEIERCIRHGQLIDSASKELMKVRKRLAVF
EERLKSKLDSLMSRHRSILQEHVLSSRGGRYVLPVKKEHRKLVKGTVLDESASGQTVYIEPAEVMTIQFELSGLRAEEAR
EELKVLSDLTALAESCDQQLLTNIETVGTYDFIFAKAKYALAIGGSNVELNDQGIIDLREAKHPLLNRPMVPLNFVIGRR
YKSLIVTGPNTGGKTLSLKTVGLLTVMAQSGLLIPVREGSLLSVFNHLAVDIGDGQSIEHALSTFSAHIRNVNEMLQFAD
HSTLILIDEMASGTDPGEGVGLSVAILEELHRRGATVVATTHFTEIKNFADVTPGFENARMEFDTDTLRPLYRLRIGEAG
QSYAFLIALKLGLPAGLIERSREITVRGLPESLPQGTQTAGTEAEPEEAPAAADTDGPQEEMPVMNVPLPPSEKEKEPEP
EPEDGADTTVPAVPGSFQLGDCVYIAYLGRTGIVFETEDARGNVGVMIQKQKFKINRKRLKMHIESKELYPDDYDLDIVF
ETKENRKKRKLMGRKHAPGVTIEIKPEE
>lcl|NZ_CP026520.1_prot_WP_042226556.1_3528 [gene=lpdA] [locus_tag=PC41400_RS17875] [protein=dihydrolipoyl dehydrogenase] [protein_id=WP_042226556.1] [location=complement(3943406..3944791)] [gbkey=CDS]
MSILESVETLVIGSGPGGYVAALRSSQLGMKTAIVERSQLGGVCTNVGCIPSKALIAESHRYELLRSFNLADAASSFKNA
QDFKQGIVTKQAGGVRYLLNHAGVSILEGEARLVDEHTAVIEHAGQEQTISFKYAILATGSRPIELKAFPVGGRVLSSTE
ALSLPEVPASLVVIGGGYIGVELGQMFAKFGTKVTILEGGEQVLPGFEADLVVPVVRQLKADGINLITGATAENVVQATD
SITLHYSINQEQHHVTADYVLVTIGRKPNTDGKLGLERIGLPVTGRGLIATDEQCRTAIPHIFAIGDITAGPALAHKASY
EAKVAAEAIAGLPSAVDYRAMPLVVFSEPELSSVGLSETEAKANAIPAVIGKASFGINGRALALREPEGFVKIVADPASG
IVMGAQIVGVEASTLISELALAIEMGATVEDLAMTIHPHPTLGEVIMEAAENAVKKMRMNS
>lcl|NZ_CP026520.1_prot_WP_042227403.1_3529 [locus_tag=PC41400_RS17880] [protein=TetR/AcrR family transcriptional regulator] [protein_id=WP_042227403.1] [location=3944988..3945554] [gbkey=CDS]
MPYPKGHKIKVRNKIVESAAQAFRTHGIHDVSVPFIMKGAGLTHGGFYAHFDNKEQLIGEACRFAISDTIALLQKAADQE
KQNPKINTVIDYYLSSYHRDKAEMGCIIPALSAEVSRSSEDVRQIFTHEVERMITFISTLAEIDTSKGSALFSTMVGSLV
LARTVNDPELSDNLLSAGRQYAKELVRA
>lcl|NZ_CP026520.1_prot_WP_042226557.1_3530 [locus_tag=PC41400_RS17885] [protein=hypothetical protein] [protein_id=WP_042226557.1] [location=complement(3945591..3946454)] [gbkey=CDS]
MTPQIMRILKKLYRADNYEYDRERSVSIYSESILSEKERDMLEQYGWSANDIVYFANHDDILQKLLSLKDNTALSQKRCI
ESFIAGVGGSYPRGRSVLSAWHKLNTLSLHRYDEQGHFACCWVCQGHDEPLFENGAYFQYCLYLGHAYTSQPHYAYLNLK
HLVNAPAVQPTVQDVQTFSTLLDLLRNAPEDETPGQFEKRLTSDKILSGDKYTKRGILDSLASVGVIPNPFLTLTPYSWT
DFGDIASSEDRLSNTKGRSDMEMPWAGWRGSLKPNEEIMTELFGDYL
>lcl|NZ_CP026520.1_prot_WP_042226558.1_3531 [locus_tag=PC41400_RS17890] [protein=hypothetical protein] [protein_id=WP_042226558.1] [location=complement(3946588..3946869)] [gbkey=CDS]
MRELIRISREVILLLSTVTCTPGTALLAPNKTGQSAQKEEAAHEQTQTAFLQFDLIRSPRIRHVMEPLFWCDEDDLQNLL
DPITRDKVSDPTS
>lcl|NZ_CP026520.1_prot_WP_042226561.1_3532 [locus_tag=PC41400_RS17900] [protein=GNAT family N-acetyltransferase] [protein_id=WP_042226561.1] [location=complement(3947160..3947687)] [gbkey=CDS]
MISITQMKLEDSFKIRDIDRSETIDLIYTCKNEVLEEIKAGHECPNWQEDDYTAIISRYEVELANGGTAFGAYDGGKLVG
FGVLANKFRGKEKDRLQIDLMYVTRDYRRQGIGSRIMDELSKAAIEKGAKYLYISSTETGSAVNFYASCGSAITSEVDEE
LFEKEPDDIHMIKRL
>lcl|NZ_CP026520.1_prot_WP_042226562.1_3533 [locus_tag=PC41400_RS17905] [protein=DMT family transporter] [protein_id=WP_042226562.1] [location=complement(3947845..3948711)] [gbkey=CDS]
MNQTTSGWIHGFAGVLIFSGSLPATRLAVADLDPIFLTVSRAAIAGLMAGALLLVFRQQRPARSDIVPLLLVALCVVVGF
PLLTALALQYVTSSHAIIYIGLLPLSTAIFGVLRGGERPKPAFWILSALGSSLVAGFALTQGAAASPVGDALMLASIIVC
GYGYAEGGRLSRRIGGWQVISWALVLSLPVMSLLSFYYMPGSWAGISTSAILSLAYVSLFSMLIGFVFWYRGLAQGGIAA
VGQLQLLQPFFGLLLSAVILHESIGWPIFVVNIGVVLCVAIARRFATK
>lcl|NZ_CP026520.1_prot_WP_042226564.1_3534 [locus_tag=PC41400_RS17910] [protein=PLP-dependent aminotransferase family protein] [protein_id=WP_042226564.1] [location=3948909..3950285] [gbkey=CDS]
MANKKNTILQIIRADIVNGLVKPGEKLPSIRMQSQRFGCSLNTIISVYRELEDQYLIYSRPKSGYFVVAKEVLLPLLQTK
RIDFASAAPDPSSIPQTDFRQCLNKAMELFETSIFTYPDPQGLLSLRQEIVKLFRRQQVFPSAQQVFIFSGTQQALHILA
GMPFPNGKTNILVEQPTFWGMLAALHTQKQTVIGIERTPDGIDWTALERHFQSNQIKFFYTIPRFHNPLGTSYTAEDKQR
LAELAKAYDVYIVEDDYLADLDTNSKADPIRAYNGASHVIYLRSFSKVMLPGLRIAAAAVPDSFIPILQWHKHSADLSTS
VLSQGVLEIYMKSGMYAHHAKQMQSLYNRRIKHLGEVSQRLLPQECLYTIPSCGGLFFSIQLPDTIDQEDMVNRLENRGV
DVLSGDRHFLPSFPKPNLLRLSIIRTDEQQIEEGIRIISEELERAPLTGRSPKPFFRL
>lcl|NZ_CP026520.1_prot_WP_042226565.1_3535 [locus_tag=PC41400_RS17915] [protein=helix-turn-helix transcriptional regulator] [protein_id=WP_042226565.1] [location=complement(3950370..3950591)] [gbkey=CDS]
MAIIINIDVMLAKRKMSVTELSERVGITMANLSILKNGKAKAVRLSTLEAICKALDCQPGDILEYKMDDNAQE
>lcl|NZ_CP026520.1_prot_WP_042226566.1_3536 [locus_tag=PC41400_RS17920] [protein=DUF2975 domain-containing protein] [protein_id=WP_042226566.1] [location=complement(3950601..3951083)] [gbkey=CDS]
MERGTTLFLKIALLIIGIPVLAMCIFLVPRIGNFAAELYPEHTYLKYLLWIDLYATALPFYFALYQAFKLLSYIDKNKAF
SELSVQALKIIKICAVIFGGLYVAGMPLFYLMAEKDDAPGFIVIGMVLIFASMVIAVFAAVLQKLLKEAIDIKSENDLTV
>lcl|NZ_CP026520.1_prot_WP_042227404.1_3537 [locus_tag=PC41400_RS17925] [protein=SprT family zinc-dependent metalloprotease] [protein_id=WP_042227404.1] [location=complement(3951328..3951981)] [gbkey=CDS]
MKIELGQQQIEVHVEYGPRKKLSLHIDPSGLVTVKAPNHTGNDVVMNAVKQHGSKILKQLQAIEAARTPMKIREYEEEGK
FLHLGRYYLLHELIETNGRTEQELQDDLKKFYFVSCKKVIGERIKIYQKQLKVTPKSFIIEESRTKWGSYSSAKHLTFNY
RLAMAPIDVIDYVVIHELCHLLHMNHDRSFWRRVGSVMKDYKEKEAFLARYGHAMTL
>lcl|NZ_CP026520.1_prot_WP_042226568.1_3538 [locus_tag=PC41400_RS17930] [protein=LCI family antimicrobial peptide] [protein_id=WP_042226568.1] [location=complement(3952382..3952654)] [gbkey=CDS]
MMKKILINSALSLSLLFSAAPAFAAVDTSTPTTCKDTIQDRYLKNRTGIFSDHFTENGTTWYIKGITSEHVSNDGGYVVY
TAHYQAHVCG
>lcl|NZ_CP026520.1_prot_WP_042226569.1_3539 [locus_tag=PC41400_RS17935] [protein=carbohydrate-binding protein] [protein_id=WP_042226569.1] [location=3953075..3954562] [gbkey=CDS]
MKKFMYAPLSLGMAFSLLFGLSGLPSNGHAAGNDLLLNPEIISGSTDRTDSGSAVLNLGDNLITNPGFEDNLASWTNWGN
TAAVTSPSFAGAKSARIASGEGGAGQIISGIPSGKTYVLSGHGSVSASTDTAIVGVDCLDANNNVLAKNTLRFNQTLYEF
KSTAFTTVPGTAKLQVYIYKNADSGANAFLDDLSLKARSGVQAINKDVKAMWVWQETTSALQDSAERSTLFQFAQDKGVN
VLLLYINKTHLTTNNTQLKQFISQAHAQGIRVEAVSGEKEWGLTDNHSIGLERITDVITFNGSAASSEKFDGVHQDIEVY
LNKEKWESNKTDVMKQALAYIDKATALMSGTGMTYAQDIPRWYDGPEYTVTYNGVDKGFYKHLIDRLDWVGLMDYTDRYS
DGQWQAVRDAQSEIDYAAQVGKKITVGLETIDPVTAGGDSITFYEEGLDDLDRALAAIKAHYDGNAGFGGVAVHSYGKQK
GNAYYSLPVKADNHY
>lcl|NZ_CP026520.1_prot_WP_042226570.1_3540 [locus_tag=PC41400_RS17940] [protein=LysR family transcriptional regulator] [protein_id=WP_042226570.1] [location=complement(3954667..3955545)] [gbkey=CDS]
MELLQLQYFLAVARLEHMTEAARSLHVTQSSLSKTIQRLEEDLGVPLFDRTGRRLRLNEFGIRFLRRAERALFELKQGKQ
EMRDLSGLEHSTLELAVTTASTLPNILRAFRKKMPDIQFHVQMLATQEMVTLLHRGEVDFCLSSPPIKGEDIECQIVCID
PILVAVPKGHRLAERTSVSLTELREEWFVGVKRGYGTRDLVDSVCNSVGFAPKYVYEGDEPARLSALVEAEIGIAFIPST
ARNSRENIRYLQVEDHELVREIALIWHRSRYISRAASEFRKVVVDYFGTNPE
>lcl|NZ_CP026520.1_prot_WP_042226572.1_3541 [locus_tag=PC41400_RS17945] [protein=MFS transporter] [protein_id=WP_042226572.1] [location=3955660..3957042] [gbkey=CDS]
MNARSWWLMISVGLGILLNPLNSSMVSVAIPNLQHVFRLDFTRVSWIIFSFYIASSIAQPVMGKASDLFGRRKIFLTGLV
VAFLASTLAPLSPNFGWLIGFRIVQSVGTSMMVGVGMAIVRMHITENQATALSVLSIFLSGAAAIGPFIGGILIHMWDWP
SIFFVNIPFAVASFLLAWRTLPKDEPPTSVTRDMSIRKWLNVIDASGVLLFAVGLVALLIGLLSAKSSGHLSLGNVIVAL
IGLLLLLAFVRHELKTTSPFIPIRTFAEYPAMTWVNVEFMLVNVLYYSLFFGIPSYLQTVRHVSGFHTGILMLSLGLCSL
AASPLAGRWIDKSGPRPALFLSAILMTFGSVWLVTLNPTSPVISVCLALIAFGISNGLNNVGMQAALFQSSPKEIIGVAS
GLFNTSRYLGTILSSLLIGTVMGGTFSFEGFRTLGIILTVLSLSLVFMSRRRGDSAQLQK
>lcl|NZ_CP026520.1_prot_WP_042226574.1_3542 [locus_tag=PC41400_RS17950] [protein=MarR family transcriptional regulator] [protein_id=WP_042226574.1] [location=complement(3957115..3957612)] [gbkey=CDS]
MADERKQIAEQLMESYFQAQVGGNMMQNRAVHNQKPGEIMVLYFISMNVEDESPGMKVSEISGKLNVTSPTVTQHINSLE
GQELVERHADPADRRVVRIRLSDKGKKYIRHINEARLNMFVGLVEHLGEKHSLLFAELLKNASDYMLKQQEFYIRSLLAD
ENDEK
>lcl|NZ_CP026520.1_prot_WP_042226575.1_3543 [locus_tag=PC41400_RS17955] [protein=hypothetical protein] [protein_id=WP_042226575.1] [location=complement(3957667..3958689)] [gbkey=CDS]
MRMRGVTYDTGFLSEGTSTHPDFDATRVSREMQIIRHDLHCTAVRVTGGEPVRLEIASQAAAAAGLEVWFSPFTNDLDQD
ELLDLLADCAERAERLRVTGAKVVMVTGAELMLFTRGFLPGETLPERLAALSERAVSPSAFAKLPALLNGFLTRAVDVVR
ARFGGPVTYAALPSEPVNWEPFDFIAVDAYNSAEVAHLYADSIKKLAGSNKPLVISEFGCTTFKGATDLGARGMFMVEWD
GAQAVGLKGSYERDESEQSRYFNDSVDIFEDAGVDTAFWCTFASHNLPYDEDPDRDFDRASYGVVRVLPERCGIAYPDMP
WEPKEVFHTIAACYKAAMKS
>lcl|NZ_CP026520.1_prot_WP_042226577.1_3544 [locus_tag=PC41400_RS17960] [protein=hypothetical protein] [protein_id=WP_042226577.1] [location=complement(3958926..3959312)] [gbkey=CDS]
MKRRFSMILVFLTLLFSISSTAAASGEEEVYSYRFTFQGPLWTFNYNLNAYYSGSGSTPSLRAALFKVHSWGEEDLDREE
RFFPLYYSPRSFASSAQFLPLVTGTEYVIKVYSTAPRSDWGGISISPF
>lcl|NZ_CP026520.1_prot_WP_042226578.1_3545 [locus_tag=PC41400_RS17965] [protein=VOC family protein] [protein_id=WP_042226578.1] [location=3959671..3960123] [gbkey=CDS]
MSKNKLLRMDNVGIVVESLDDSISFFEEIGLKLEGRATVEGEWAGRVTGLGSQCVEIAMMVTPDGHSRLELSRFLTPPPI
SDHRTAPVNALGYLRVMFTVEDIDEMVSRLTKHGAQLVGEVVQYEDSYRLCYIRGNDGLLIGLAEQLGNK
>lcl|NZ_CP026520.1_prot_WP_241688907.1_3546 [locus_tag=PC41400_RS17970] [protein=alpha/beta hydrolase] [protein_id=WP_241688907.1] [location=complement(3960279..3961229)] [gbkey=CDS]
MKKRMGKPLRILLKYLGAIAIVIVLFLAVVFIADKVASKSDQGKIEPYGQFVTIDGKKMNVLIQGKGQETVVLLPGYGTA
APALDFKLLIDELTPYYKVVAIDPFGYGLSDETEKERTTENIVSEIHEALQQLHITRYTLMGHSIAGIYGLDYVNKYPNE
VSAFVGIDSSVPTQPGMDVKFPLTMLKFLKQSGLQRFLVSVGSDPYASLLFDDKTKEQMKMIMNKNANNNTMLNEMKNIY
SNFKGAQNLTFPKNLPLLLFVETNNTSVEGWIPLHEGQIKDSVHGKVVTMDGTHYLHHTKFKEIVESLREFQAEIK
>lcl|NZ_CP026520.1_prot_WP_084706507.1_3547 [locus_tag=PC41400_RS17975] [protein=alpha/beta hydrolase] [protein_id=WP_084706507.1] [location=complement(3961372..3962361)] [gbkey=CDS]
MKVTQQDKKKTRNGTRTRKVLTILLKIIGAILIAIMLFVAIVFTVNKISNHSEQKRLEPYGQHVAVDGKNMNVTIQGQGE
ETIVLLPGFGTAAPALDFKPLIEELSPFYKIVVVEPFGYGLSDLTEKERSTENIGSEIHEALQALHIDRYILMGHSISGI
YGLDYVNKYPSEVSAFVGLDSSVPTISEKRFSSSEIETVKLLKKSGFTRLLMKPDTDVYAELPYDDQTIEQIRILMYKNM
VNPNQLNEIEHMYANFKRAENSTFPKNLPVIFFIEANHPAADRWIPEHEKQIKDSVHGKVVLLEAGHYLYRSHSKEIAEN
FREFMKEIK
>lcl|NZ_CP026520.1_prot_WP_042226581.1_3548 [locus_tag=PC41400_RS17980] [protein=HAMP domain-containing sensor histidine kinase] [protein_id=WP_042226581.1] [location=complement(3962579..3963946)] [gbkey=CDS]
MIKSLYIRVVLTFLVSVIAGTVIAFFVSTWIFEDKLNENAQINLRNFGQDIVRIYKTLPLREADSFVSEMKQLDSYYIRI
YEATGQFQSYGKLNGHKPAVVTEEQLKKVIDGGVVQDTPNGIASVLLGLPLKTETGTKAMFLETLAPPSASFVVKWAWIF
ATCSLIAGSLLILVASIFLVKPIKKLTKATKRIATGDFNVKLNIKQKGELGTLARSFEDMMHDLQQLEQMRRDFVSNVSH
EVQSPLTSISGYAKALKQVNIADHERSRYLDIIIAEAERMSKMSDSVLKLSLLESQSQQLRLSTFSLDEQIRRVIVAIQP
QWSARNIRFELNLKAVSLTADYDLLNQVWTNLIANSIKFSKDGDVISVTILQDIKNVIVRISDTGIGISLEDQKRIFERF
FKADRSHSRKYSGSGMGLAIVKQIVSLHQGDIRVESEPGRGTTVIVTLPITTPTE
>lcl|NZ_CP026520.1_prot_WP_042226583.1_3549 [locus_tag=PC41400_RS17985] [protein=response regulator transcription factor] [protein_id=WP_042226583.1] [location=complement(3963943..3964617)] [gbkey=CDS]
MPTILVADDDANIRELVCLFLRNDGFSTAEAADGKEALAVYTSTHIDLVVLDIMMPIMDGWTLCKELRRANPDLPLLMLT
ARGETWEKVKGFELGTDDYLTKPFDPLELTVRVKALLKRYRIGSTQTIQFGSVILDRQTYKVMRGTESLTLPLKEFELLY
KLAGTPGQVYTRGQLIDQIWGIDYAGDDRTVDVHVKRLRERFASTPEFRIETVRGLGYRLEVCE
>lcl|NZ_CP026520.1_prot_WP_042226585.1_3550 [locus_tag=PC41400_RS17990] [protein=winged helix DNA-binding domain-containing protein] [protein_id=WP_042226585.1] [location=3964978..3966099] [gbkey=CDS]
MKDQAASGSVLSRHVLNRTLLQRQLLLSRPDLSVPEAIRHLVGLQAQASNPPYIGLWTRLEDFRHEDLSQLLLERQVVRI
AMMRSTLHLVTAHDCLGLRPLLQPVLDRGLKGSYGRKLAGLNIGAIAAAGRTLAEQEPRTFSELGSLLGNQWPDTDPAAL
AAVVRTLVPLVQVPPRGIWGVGGLAAHTPAETWLGKPLQASTGPDEMLLRYLTAFGPATVQDMQVWSGLTRLREAVDRLR
PRLMTYRDEHGSELFDVSGLPLMDPDSLTPLRFLPEFDNMLLSYADRSRIIAEEHRSRVFTVNGIIRATVLVDGFVSGTW
KVERSRGSATLVIEPFKPLSTEYRTALAEEGSRLLRFIAEDAETYDIRYVKPE
>lcl|NZ_CP026520.1_prot_WP_042226586.1_3551 [locus_tag=PC41400_RS17995] [protein=OsmC family protein] [protein_id=WP_042226586.1] [location=complement(3966302..3966691)] [gbkey=CDS]
MVISTSGSEAYITELYNGTSKIDSDVTKEKGGSGKYFRPHDLIESGYASCLNITTRMVLDSMNLSYDGVTVKVELDRSNE
EKTIFKYQIDISGSIDKETREEVLKRVANCPVKKTLSKQLEFINDCNII
>lcl|NZ_CP026520.1_prot_WP_042226587.1_3552 [locus_tag=PC41400_RS18000] [protein=hypothetical protein] [protein_id=WP_042226587.1] [location=complement(3966858..3967430)] [gbkey=CDS]
MSDNELILQQKALQTEAGKIAEELKLNDLLSQAGEPVRVGSSALGLMVWRDLDITVICSRLDQNVVAQIASEIMVRTGVR
ELKFINDSGTWNTDPTYPDGLYLGLKYRSGQDKDWKLDIWFIDEPERQPDLGHIKTMPERLTPERRHSILRIKSIWAERA
EYGKTVRSYDIYTAVLDDNVHTPDQFLEWL
>lcl|NZ_CP026520.1_prot_WP_042226588.1_3553 [locus_tag=PC41400_RS18005] [protein=methyl-accepting chemotaxis protein] [protein_id=WP_042226588.1] [location=complement(3967683..3968552)] [gbkey=CDS]
MNNVTEAQTQHELIRAFVKVAPLLSSLNHDDITIGIYDTEKLIINIPGKTFSLNVSPGDPLMEGDIVTNAIRNNTPLTAV
VPKELFGFPLAARAIPLHDEQGRVIGGVGMGTSLEKANKLFEMAESFSAIVEQTTASIGDISQSVSNLTDRVTDVTHQMQ
DVSSSAQQIGKISAVVKEISDQSNMLGLNAAIEAARAGEVGRGFSVVADEIRKLATSSKENVDQINGITKNIQELLLRLN
HAFSDIYTLTDTQSKAIREFSATIQEISTKAEDLAQVAEATLYSSESKQ
>lcl|NZ_CP026520.1_prot_WP_042226590.1_3554 [locus_tag=PC41400_RS18010] [protein=aldehyde dehydrogenase family protein] [protein_id=WP_042226590.1] [location=complement(3968574..3970028)] [gbkey=CDS]
MDFQQLGKQFIGGEWRDGKSTKSLPDINPYNDAVIATFQIANVADIDCAYQAAYKAKLEWDKVNPYAKRDILENAVKYIE
AHEEEITSIIVQELGGTRLKAAFEIGLVKNMIKEAATFPLRMEGKILPSVEDGKENRLYRIPVGVVGVISPFNFPFFLSM
KSVVPALGAGNGVVLKPHEDSPITGGTLIGKIFEEAGVPKGLLNVVVTDIKEIGDAFVEHPIPRVISFTGSTKVGSYIGQ
LAVKNFKKPLLELGGNSAFIILEDADIDYAVQAATFSRFTHQGQICMSANRILVQQTIYQDFTDKFRDKVSTLTTGDPSD
PQTIIGPVINNRQAETLEKLIASGIEQGASALVQGKITGRMVEPTVLVNVQPDMAVAQEELFGPVVCIIPFETEEDAIEI
ANDTRFGLSGAIHTSNLERGVEMAKKIHTGMIHVNDITINDEPIVAFGGEKQSGLGRLNGEWSLDEFTTLKWISVNYKQR
KFPY
>lcl|NZ_CP026520.1_prot_WP_042226592.1_3555 [locus_tag=PC41400_RS18015] [protein=right-handed parallel beta-helix repeat-containing protein] [protein_id=WP_042226592.1] [location=complement(3970238..3971782)] [gbkey=CDS]
MQIGKTFITSLSLTAVLGLGLGIAQSPSVSAAGPDYYVAANGNDSNPGTLSSPWKTLQYAANQAGPGSTVYVRGGVYNEK
LKITKSGSAQQGPITFSSYNSEAAVIDGTGLSVSGTEGLIDLTDASYVNIKGFEVRNYTTAKSNVMPVGIYVHGAGGFIN
LSGNKVHDIKNTAAPVGTDLLGRDAHGIAVYGTKAPQSIHDVTIDGNELYNLVLGSSESLVLNGNVDTFAVTNNTIHDND
NIGIDVIGFEGKAPDAAYDQARNGVIRGNRVYNISSNNNPSYGKKLPNNSNSAGGIYVDGGKDNLIERNYSYGNDIGIEI
ASEHAGKATSRITVRSNMIYNNRLTGLAMGGYDKQRGATANCVIVNNTFYKNDTLGDGSGQLYVQYDTQNNVIKNNIFVA
SSSNVLIYNEYTKNSGNIVDYNLYFAPGGSTNAAWTWKNVTYNGFAAYKKGTGNDAHSLFADPSFVNAASSDFHLKPGSP
AIDAGSTEPDIIGSLDIDGDPRVRGAAVNIGADE
>lcl|NZ_CP026520.1_prot_WP_042226594.1_3556 [locus_tag=PC41400_RS18020] [protein=carbohydrate kinase family protein] [protein_id=WP_042226594.1] [location=complement(3971890..3972849)] [gbkey=CDS]
MMHTGSKSRPVVVVAGEINADLIVSGRDVIPEWNREKIVDGFQTVLGSSSAITACALASLGVDVRFVGVAGQDDLGAFCI
GELQRMGVNTDGVTVLPDVKTGVTLSFTTPSDRGLLTYPGSIPMLVPEYIPENILREADHLHFGSYFLQDGMRPHWKNLF
REARARGTRTSFDAGWDVRKEWDSEGIRSLLEETDLFMPSEDEALHIFAADRIADIREKLGNAAGRVAVKQGSKGASLLA
PDGSVVTAAPFRVSPRDTTGAGDCFNAGIIYASLQGLKGEELLRHGNACGALSTLGVGGTGNLPTTDAVGRLLKEQAAV
>lcl|NZ_CP026520.1_prot_WP_042226595.1_3557 [gene=melA] [locus_tag=PC41400_RS18025] [protein=alpha-galactosidase] [protein_id=WP_042226595.1] [location=complement(3972846..3974147)] [gbkey=CDS]
MTKISIIGAGSAFTQEITTDILLIEGIEGGTLALVDIDGERLEIARKLVARIIEKTGKKWNVIASTDRREVIGGSQFVIN
QIEVGGLQTVRYEYEIPLKYGVNQCIGDTLGPGGLFKTLRTLPAWMDIVRDIEDLCPDATILNYTNPMSAVTLLTSRITT
LPVVGLCHSIQNTSSQLAKYAGVPYEQMQWKAGGINHMSWFVELTHEGQDLYPLLLEKIQDPELLKKDPVRFDAMKYLGA
FVSESSGHFSEYIPYYRKRQSLIDRHCNSGYNGATGYYAENWPIWRKENDEKIIEQLEGKRPLELQPSNEYAAIIIEGML
KNEPKVIYGNVPNTGLITNLPSDGIVEVACMIDRNGIHPCHFGSLPEHLAALCRSNMAFFELAAGAVLHKDKEMARHALM
VDPLSAAVCSLEEIGEMFEELYEAERDFVPVLK
>lcl|NZ_CP026520.1_prot_WP_042226596.1_3558 [locus_tag=PC41400_RS18030] [protein=carbohydrate ABC transporter permease] [protein_id=WP_042226596.1] [location=complement(3974171..3974995)] [gbkey=CDS]
MQRRNRWLDGAILIVLAAAGCLMLLPYLWMVLSSLKSNMEIVSGSHTLFPQEASLEGYRTVLVDAPFVRWLLNSAVTSVV
ITAATLFTSALAGYIFAKHEFKGKKILFLLILATMMIPFQVVMIPTYLITAKLGLVNQLLAIILPSLVSSYGVFLAKQFI
EDIPQDLLDAARIDGAGELRLMVRIITPLIFPMLSALGIFTFMNSWNNYLWPLIVLDDTRKMTVPLALVYFNGTHTVNYN
VVMSAAVLITIPVIIVFLIFQKQFIKGLTMTGMK
>lcl|NZ_CP026520.1_prot_WP_042226597.1_3559 [locus_tag=PC41400_RS18035] [protein=sugar ABC transporter permease] [protein_id=WP_042226597.1] [location=complement(3975011..3975946)] [gbkey=CDS]
MNLEPAAGRTRKIPPAAQHKGRKNARAFVYLSLIPILLLFGAFAFLPIGWSLGLSFFKYNPLSSNTTFIGVENYARMLGD
SVFLKSLWVTVKFVVVAVAVNIAITLLIAVAIQRLQTGWLRNLMRTVFFLPTIAPLAGTAIVWSTMFNYNHGLFNMLLEK
LHMAPVQWLSDPHYALYSVIIMTLWADIGYNIVLFIAGIDSIPGLYYEAALLDGANRWHSFRYLTLPLLRRTMLFVSVTT
VVSYFQAFAQFQIMTKGEPFNETRVIALHIYEMAFSNSNMGYASAMATVFLMIILAVTVLQLKWGRTDWEH
>lcl|NZ_CP026520.1_prot_WP_241688687.1_3560 [locus_tag=PC41400_RS18040] [protein=ABC transporter substrate-binding protein] [protein_id=WP_241688687.1] [location=complement(3975963..3977099)] [gbkey=CDS]
MNAYKDLIAAYQAKHPEVEIDYQSFPYDVYPQKLKASFIAKNPPDIAEMFGTWVPVYSTNGLLAEIPDGERLRDGYYKAP
LGGYSLNGKLYGLPLEFNIENGGMLIHPKMLKEKGIEGPPSTWEELVESARKLTVRDGKGIKVKGFDFVSGDNITFTFLS
MILQQNGKYWGENDHVNFRTPEAKKAMQALVSLIKDDKVADLTTFGGELDISDYFFQGSSAMTYRGPWSIAAGLNTYKAE
DFEYVPVPSFTANPPSYAAESGWGLVASQKSKQRKAAVDFIEFVSSNENLLQWNLSTFTVPAKKEVAENPEFLKANPFMK
TSLDILTLGEWIGPITDRDYFFKVINDHFQLMASGQETVEAGLRGIEKSINDNQDQHK
>lcl|NZ_CP026520.1_prot_WP_042226600.1_3561 [locus_tag=PC41400_RS18045] [protein=LacI family DNA-binding transcriptional regulator] [protein_id=WP_042226600.1] [location=3977490..3978533] [gbkey=CDS]
MPTLKDIADRVGVSISTVSRVLADDISRASLSKTRRKIWDAAQELGYTIKLSAQEPVKRIGCIVSTMQGRRYHPYFSVIF
EGIEKELALHGYQIAFTYTQEDLQKPEILQQVIHKDQTGGLIIIEGIEKNLYDYIRKHVEHIVGIDVSDPDIPTISYDRV
EAARTAVKHLTDQGHRDILFIGGTGLSGEFQREKRYRGYKLALEQAGLTVQPQYVLNAEWEPDQAYRLVLNYIDEHRGHL
PTAVFAASDIMAMAAMRAISEKGYGIPQDFAIIGFDDNEPSRYTVPPLSTIHIPTLEIGSVAARTVLQEMQTPYPLPVKI
LLPFQPLFRQSSDRRLQKLQDNNNETR
>lcl|NZ_CP026520.1_prot_WP_241688688.1_3562 [locus_tag=PC41400_RS18050] [protein=DeoR/GlpR family DNA-binding transcription regulator] [protein_id=WP_241688688.1] [location=3978860..3979663] [gbkey=CDS]
MRLLYQEERLLKILDYIKKHHAMSVMDICSLLHVSRDTARRDIVKLVQEGVVIRTHGGVALPQLQKEIATYQERLIDETD
SKRKIGKLAAKLVQDHETVILDVSTTVQSAAEQISAKHITAVTHSIDIVGILSGREDLQVYVLGGYLHAKNRLLYGPSVI
EKLRELRADKAIIGATAIRADGLYFPYEEDVSVKKEMARRSDQVIVVADSTKFTSKSVYRLDFDFVDILITDQEVPIDIK
EVLHNKNIALIQCEEKDNYEEDDKDEY
>lcl|NZ_CP026520.1_prot_WP_042226604.1_3563 [gene=nagA] [locus_tag=PC41400_RS18055] [protein=N-acetylglucosamine-6-phosphate deacetylase] [protein_id=WP_042226604.1] [location=3979653..3980798] [gbkey=CDS]
MSIKILHHVNVIQEDNSTLSVTLWIRDGIIEKIESGLATLPEGDYEYMDGQGRLLIPGMIDVHIHGANGFDMMDGSETSI
QEVSRACAATGCTSFLATSVSSTIEDLIAMIRSVKRVAGHEVGAKIAGLHLEGPYLNPKRKGMQNERFLRHPDLAEMKEI
FNEADGLIKMVTIAPELPGGMELISFLKETGVIIAVAHSDATYEEAKQAFEAGASHVTHCFNGMRPIHHRDPGLVVAAFE
EEHVSLQAIVDGIHLHPAIVRMMHRLKGPDDMVLITDALQAMGLGDGKYMFGGHHVTVSEGVARLEDGTLASSTVTMNEA
LRLTVDTGISLTDAVKMASSSPARLLGLANKGKISVGFDADLVLLDEQFQVQWTMIKGSIV
>lcl|NZ_CP026520.1_prot_WP_042226605.1_3564 [locus_tag=PC41400_RS18060] [protein=acetylhydrolase] [protein_id=WP_042226605.1] [location=3980892..3982286] [gbkey=CDS]
MRLFEALLIGSNLFLLVVLLFMRNRKIGLGAGFAAICALSAHLIWEGYRWQLAPSYLVSVILFILVAIRIYRNPKSVNLR
KKTRYIGYSVLTVLLGFSVFLAQAVPILVLPKPGGEAAVGTQLFHFTDSTRQDSYPNLPKNDRELMVRVWYPAKPQPGKG
ELLFPEDPVQFKTYKENFSGVFGLPAFILDYWKYTRGNSRLNAELLPASAPYPLVLISHGMGTGSVLHVSQAENLASHGF
IVAAIDHTYSTAATLFPDGRITGFETKTPANDFYAYAKRVGAVWAADIQFVLSEMEHLNQDPESSLKGMIDVNHTGIMGH
SFGGGTAFNVTYSSDKIKAGIDMDGTLYDLEDKGQMPKPFMFLNSDKRSTLSAAIKNNEVPDSIRAQLVKEQQAMDSIGA
RNGQVIHIRGTDHFNFTDLQFYTPMLKYTGMTGDIDGYRGAALVNSYVLDFFTRHLKKNGGKPL
>lcl|NZ_CP026520.1_prot_PC41400_RS18065_3565 [locus_tag=PC41400_RS18065] [protein=ATP-binding protein] [pseudo=true] [partial=5'] [location=complement(3982301..>3982547)] [gbkey=CDS]
GNQITFTIRDSGPGFSSEELERVFELLYRGESSRNRSTGGSVLGLTISQKNCQAARRRPCR**PSRGGSPAERLAPGRWS
GL
>lcl|NZ_CP026520.1_prot_WP_042226607.1_3566 [locus_tag=PC41400_RS18070] [protein=RNA polymerase sigma factor] [protein_id=WP_042226607.1] [location=3982889..3983446] [gbkey=CDS]
MEDKEIIQLYFARNERAIEETAKKYKHYCVRIALNILSSVEDAEECVNDTFLGAWRAIPPKSPAKLSSFLGRITRNIALD
KYDYHTAKKRSKEFETILDELNDCLSSSDNVESRYEEEQMAEAISQFLKTLNEDHRNIFVRRYWYSDSMSDIASRFSISE
SKAKSVLFRTRKKLQTYLKKEGYSV
>lcl|NZ_CP026520.1_prot_WP_241688689.1_3567 [locus_tag=PC41400_RS18075] [protein=hypothetical protein] [protein_id=WP_241688689.1] [location=3983443..3984348] [gbkey=CDS]
MNNKDLFKTIGYIDDQLIEKAHTARRATKKSYALWGSVAACVALVSGLALYESPPIPQPATPQIFQQSLPASPTGIRKLL
TYNGYRYAFVDDGAPFKFTGTEPVKSLGTLVADTGPETGKNTEGSNVEKDYTATFAIGGKLYEIPAYPSHFRIAVQYEQN
FYLAEIVAKVNDSAISAKEYLDMSKLKEHVRDIHILNHAGNDVLKKMTDSASVKSIVEGLYTAKTADLTNREFEGIAEAQ
SQGKSYQLKFNLKDGTGMAMYIIPELKIVSMGDAYYQLPDRFLDLSGDLFKNLKQDALPLY
>lcl|NZ_CP026520.1_prot_WP_042226608.1_3568 [locus_tag=PC41400_RS18080] [protein=lytic polysaccharide monooxygenase] [protein_id=WP_042226608.1] [location=complement(3984508..3985533)] [gbkey=CDS]
MNLALPNGTLLQKLIAACGMLLMLALLTVLFPETGSAHGYLDVPASRALLCKNNVNKDCGGASYEPQSAEAGKGFPNGGP
ADGHIASAGNTFPKLDEQTSTRWSKVPMTSGPQAFKWQLNAAHATTTFKYFITKPGWNPNAPLTRASFDLTPFCQVDLNG
ASPSNYYTTNCNVPERTGYHVILAIWEVSGAPTAYVNVTDADFGGGSALPAPANLASTAVTDNSVSLSWASVAGAASYQV
YRDNVAVGTTTSTSYTNTGLSSGTSYSFTVSAIDSSGKASPSSSVVTVKTTGGSTTTYPAWSATTVYVGGNKVSYNGVNY
EAKWWTQGETPTGTNVWKVIP
>lcl|NZ_CP026520.1_prot_WP_042226609.1_3569 [locus_tag=PC41400_RS18085] [protein=ABC transporter ATP-binding protein] [protein_id=WP_042226609.1] [location=complement(3985824..3987560)] [gbkey=CDS]
MDLLRNITAGNPRALIKPVLYTTLANLIGIVPFALLVEAARLIFEPFASPGTALQTERLWWVCGGMAVSLILMYASEVPA
YRSQFRQAYSAAAEGRTRLAEHLRKLPLGYLNKRDPGDLANMMMGDFTMVEHGISHLVPQMIGALVMPVLALAGLSFLDW
RMALALFAAFPLAVALVMLTSGIQHKLGANHMRAKIDAANRLQEYLNGIRVIKAYNMTGESFARLTLSFRELMRQSIRIE
GLLGPIVLSAIACIRAGLTFMIMVGVHLLLGGSLDLIVFVSFLIVGTRIYDPLTTVLVNYAEFRYHEQAGERIVRLLQEP
VMPGKRQPPAAYDIELEQVTFGYNRQTVLREVNLSIPAGTFTALVGPSGSGKSTVLRLIARFYDPAGGKVLLGGNNIRDM
EPEALLSKISMVFQDVYLFQDTIAANIRFGRNNATMEEIEEAARQACCHDFIMKLPNGYDTLVGEGGSTLSGGEKQRISI
ARALLKDAPIVLLDEATASLDPENEAVIQEAIDRLVQGKTVIVIAHRLKTVRSADHIVVLDNGSIVEQGSHSNLLAAKGL
YARLWDIQQQADTWKMIE
>lcl|NZ_CP026520.1_prot_WP_042226612.1_3570 [locus_tag=PC41400_RS18090] [protein=ABC transporter ATP-binding protein] [protein_id=WP_042226612.1] [location=complement(3987529..3989361)] [gbkey=CDS]
MSNKSGVARLLQIAGEKRGLLAASGILSALSAIGQLIPYISVYFILKELLVHAAHPAGADGPFMVRWGIIALLGLIGSLV
LMYAGGLLSHIAAFRILYGLRIKLSSHIGKLPLGWLNGTSTGAVKKTLEQNVEKIETFVAHQLPDLIHVAVSTLLMITVM
FYLNVWLAFACIVPILVGFGVQIVLMTGPETKKNVKKYHDALERLNGSAIQYVRGMPAVKVFGQTVHSFRKFYADMIYYR
DYCVKYTDQFQNGFLIFKVILVSFAAFVLPVGVFLLSRDPGNAAFASVLLFFLVMAPGVSAPMFKIMYLMSTLRDIGEGV
ERIDRLLAEKPIPEPERPVKPVQYDVTFDKVSFSYEAKAGSGEALSNISFTAGQGQVTALVGPSGAGKSTVANLVPRFWD
VNDGAIRIGGVDVRDMTVTDLMDTVAFVFQETFLFYDTVYNNIAVGRPDAAAEDVYAAARAAQCHEFIGRLPQGYETRIG
EGGVYLSGGEEQRIAVARSILKNAPVLVLDEATAFADPENENKMQLALKELIRGKTVLVIAHRLHVIRDADQILVLSGGR
LAEQGKHDELVAANGLYAKMWSTYTDSERWHIGMKKEATNDGFTAQHHGR
>lcl|NZ_CP026520.1_prot_WP_042226614.1_3571 [locus_tag=PC41400_RS18095] [protein=ABC transporter substrate-binding protein] [protein_id=WP_042226614.1] [location=complement(3989402..3990364)] [gbkey=CDS]
MIRSRITLPFWAAALFILLLTGCSPAEKTASPEADSPSSGTAAQTTGEKEAFPRTIQHAKGSLEIGKKPEKVATPYIVYL
DYLTVLDEPPHASQGMDMIGSFPNLKKRLEGKQVLDLGREASMEKLLAASPDLIISEDENKFTEYNKIAPTVIIPHSEDW
KAQLRIIAKVIGAEEKAESVIADYEKKAAEYKEKLASKRGETVLAAMFSGKEFTSFNMSRMRVFYEDLGLTPVEAFKQGG
NFNLEGLAEADPDYLFVVNNYVQPVAGGVMASMKDNPIWNRLKAVKNNRVFVLDDSSVIGPMPLGKIVAIDEIMKAMSGQ
>lcl|NZ_CP026520.1_prot_WP_042226615.1_3572 [locus_tag=PC41400_RS18100] [protein=AraC family transcriptional regulator] [protein_id=WP_042226615.1] [location=complement(3990558..3991541)] [gbkey=CDS]
MTESVHDNVNQFFDQMDFFYRSLNRTENMERSDSAGEGFALRIVPRPELIIVLTNCTLHKNTQINFRFDIPMVELSICLE
GEGAGQVNGLEVPLCPNMVNLGLMGQGSGEFCFVGGRSLVTASIYIPVSSFNYYLENIDQGRTVDFQRLLGAKSFRQYRQ
AINPAMLQLVRQLVHMPYSQAIRNIHLECKVLEMLSLSFQQLLFENEPGPSAFSMTRSELEKIRRAREILLERMECPPSL
LELSRLAGLNDFKLKAGFKKVYGTTVFGYLREQRLEKAKQLLLSGRFRVGEAAAYVGYTNASYFADAFRKHYGVNPGELL
RSSSTVF
>lcl|NZ_CP026520.1_prot_WP_084706509.1_3573 [locus_tag=PC41400_RS18105] [protein=DUF6138 family protein] [protein_id=WP_084706509.1] [location=complement(3991707..3993377)] [gbkey=CDS]
MDMDHTPETILDEMKRQIDKWAAYISDKDADQIVKRTSLQVGIHDYALLEYANGRTSVSDNNLDFTVPRAAARRGDSAET
LTEELVRERIVPELAAYIGQTLDVLTPALIDYHFTFTGKFQTREGEVCVRIYKYVDEAKKNRLLERISFYVAKKLEAGEY
PTKPLETFFLSRHLLDERLFPDVDAARIISLFEKIQQVNKGNKHLAEHRNCLVVALRNWAEDQWLPRYFDQVGTEFQKQF
KKKSGARLENTEQGPIELILYTAILILKYEPSYSRSTGLDILNCAIELGSARAKRLTMEGSGAFAKEDVYSRDELVECTA
NDVFAEVTLVIKQETEESYGRALRFLTRLLSLGFPKSYQLKLKSGVKQWLPVKGLAKSGTHRFFAQALEYRNLHPLLEEY
ARAAMEPFEWYADTEGEKNCMPGSYAVFGLGLADRAYFPLVEEYMGKVDEEHQYVQNHFTVALAERHGVNSETIPTLVKC
MLHSTDSLKLKIKADMEDDSHLRVLLEQVRDLQNYEVEHIHYLIWGGTDKLKKMAAKAEGDRRQWLSELLQLSGHR
>lcl|NZ_CP026520.1_prot_WP_129112508.1_3574 [locus_tag=PC41400_RS18110] [protein=DUF2935 domain-containing protein] [protein_id=WP_129112508.1] [location=3993707..3994525] [gbkey=CDS]
MADLFVSRSLEEIRFWSRIMKEHSFFLRLGFRCEDTQLINEATQFYRTFEAIEQKALSFPEETDPATIYAFNQEVHTAVS
HIWVFKRKVLGLILTCRLPGQTNFPLLVDHVSREARYFRNRLEELNQGKLEPLPDAIIDENVFFLRIMADHAKFIGHLLD
PSERKLVDQARDFSHDFDQLLYQARDMDSMRPQSQTVPLLNQFLDQNRVSVASLKDFKKTARDLIEQCKIKSIIHPLLAD
HVFREAERFLEIIELFERSLTGISSSNRVLHH
>lcl|NZ_CP026520.1_prot_WP_042226616.1_3575 [locus_tag=PC41400_RS18115] [protein=class I SAM-dependent methyltransferase] [protein_id=WP_042226616.1] [location=complement(3994639..3995415)] [gbkey=CDS]
MCSKPSTNHSNSEVYRVKPYFNVKPEEGVLSLLMPQPGELILDVGCGSGDLAAELTAAGANPTGIDLSEEMVKRARQKYP
ELNIQVEDVCHYRTEVPFDAVVSNAVLHWVKDAPAVARSIWLALREGGRFVAEFAGSGNVALLTAAMEQALEEHGYAWAG
RNPWYFPTIGEYASLLEQTGFRVTLAQHFDKPTPLKGDMGIRKWLDSFSEYFFRDVTPADKASIFRAIEAKVKPYLEREG
QWMVDTSRLRIVAFKKSV
>lcl|NZ_CP026520.1_prot_WP_164977140.1_3576 [locus_tag=PC41400_RS29875] [protein=hypothetical protein] [protein_id=WP_164977140.1] [location=complement(3995548..3995709)] [gbkey=CDS]
MDAQSPQLSGAEKTAENPTDPPQAPTADTGRDRMEEITLLNLSNVFMEAGQAV
>lcl|NZ_CP026520.1_prot_WP_042226618.1_3577 [locus_tag=PC41400_RS18120] [protein=hypothetical protein] [protein_id=WP_042226618.1] [location=3995885..3997108] [gbkey=CDS]
MDITQDFVDERFKTFVLDRFCENRGRIQTSDVEGIVSLQLANLRFSSLKGLEHFASLEELDCSFNHLTELDISKNANLTF
LECNGNKLHVLDTGHNLKLKKLSCNNNEIYSIDLEANVQLEAVDCSFNRLRHLDVGRNPMITKLICSWNMISDLHVEQNP
HLQELNCSYNSLFNLELKRNASLMYLDCGSNYLIEMNIKDGAGLVEIRCNNNHLKHLDVSQCPVLESLRCFQNHLTSLDL
NHNPLLNELYCSENKLTELNISCNPMLKRLDYADNLIVEPDHTVKGMGTFRYDGSLCSYQTALLYKDKELSVRADVPAKA
DMKRLSPVIKKTWEQLDQLHEQAADLIAHTHPDEDVTELVLSELEFDKDATIRIGYDAGESPAGHLYIYAVFNRKHELSS
QLVYETY
>lcl|NZ_CP026520.1_prot_WP_042226620.1_3578 [locus_tag=PC41400_RS18125] [protein=SMI1/KNR4 family protein] [protein_id=WP_042226620.1] [location=complement(3997269..3997712)] [gbkey=CDS]
MKEALMERLKAFLHRKDNSTLVGSPASQEEIAHAEQQLNVSFHEDYSHFIRTFGGAYAGLAVHAFSNGSSLGNETVVDLT
LEFREQFKGLPFAEVLRTGYVISMDGSGNPILINQSGQVVIGDHDTGEMELIADSFEALIEEQFCEW
>lcl|NZ_CP026520.1_prot_WP_053228632.1_3579 [locus_tag=PC41400_RS18130] [protein=isochorismatase family cysteine hydrolase] [protein_id=WP_053228632.1] [location=3998087..3998716] [gbkey=CDS]
MPFKPHPNNTAIVITDPQNDFMSPGGRGYHLTKGTIERNNTLQNLELLMRTAMNKGYQLFISPHYLYPHDYGWQFTGNEQ
NMLLGLRMYQVANAYTPPSFGADFVHSFKPYIFDRRTVIATPHKVASPQTNDLEYQLRQHRKEKIIIAGVLSNICVESHM
RHLIETGFETAVVYDATATISEEDFQAAVTNYKAFASATWTTQEAISKL
>lcl|NZ_CP026520.1_prot_WP_042226622.1_3580 [locus_tag=PC41400_RS18135] [protein=serine hydrolase] [protein_id=WP_042226622.1] [location=complement(3999020..4000042)] [gbkey=CDS]
MPTTYWPTAEWQAVDPALLRMDPEKLAELEPVIQSEYGNINGVVIVRNGAIAHESYYNGCGPDNTHHVASVTKSILSALI
GIAIDTGSIKHVDQKVLDFFPDYVPDASDSQKREITLRHLLTMTAPYSFADWHEPLDKMCMQSDWVKYTLDMLGQKGEIG
TFKYSTAGAHLLSAIITRSTGKSAREFANERLFNLIGMKEIPDYEMKSFGFEDLFGEKVRGWVQDPNRNSTGGWGLTLTP
RDMARFGLLYLNRGLWNHRQIISGAWVNESTAMNANNYGYLWWLREEEGVYAYSALGDGGNVICCIPEKDLVIAITSEFV
AHPRDRWTLIKDHIIPAAIG
>lcl|NZ_CP026520.1_prot_WP_042226623.1_3581 [locus_tag=PC41400_RS18140] [protein=MerR family transcriptional regulator] [protein_id=WP_042226623.1] [location=complement(4000080..4000904)] [gbkey=CDS]
MLSIGEFSKICEVSTKTLRYYDEIGLIHPDEINPENGYRYYSIRQLKKMLLINRLKSYHFSLEEIKAMLELEGDQSEEKL
CSALTRKRKEIQDKLTAFEYTLNQISTDILNVQKGMPILYHLDRIEVELVETQPVNILYTRQMMSRDDLVRGYENYFGKL
YEKIAVEKLTLLGKPMTIYHSSEYNPAGNDTEFAIPVGEFVKGTRDMPGGLCAKSVLQGAYSDLTSVYARLMEWIENEGY
KVVRSPYEVYITDPSQAKDSGDIVTEVYFPVKKK
>lcl|NZ_CP026520.1_prot_WP_042226625.1_3582 [locus_tag=PC41400_RS18145] [protein=helix-turn-helix domain-containing protein] [protein_id=WP_042226625.1] [location=4001214..4001570] [gbkey=CDS]
MRETCVPTGVALKDTGFGYTLSLISGKYKMIIIYWLAENKVMRHNELKRSIGSISFKTLSIMLKEMEADGLILRKEFPQV
PPKVEYSLSERGLSLLPLLSMMCKWGENNWPDSEAAQR
>lcl|NZ_CP026520.1_prot_WP_174446785.1_3583 [locus_tag=PC41400_RS18150] [protein=NAD(P)H-dependent oxidoreductase] [protein_id=WP_174446785.1] [location=complement(4001650..4002210)] [gbkey=CDS]
MKTLVIVTHPSLETSVINKRWIEELEKYPEKYTVHDLHKIYPDGNIDVKREQQLIESHGNLVLQFPVYWFNCPPLLKKWL
DDVFTYGWAYGSNGDKLRNRKAALAVSAAIKEEDYREEGRYRYTLQHLLSPFEITFLYCDADYRSLFAYYGTETESGENV
PGSENESPANGLEKSAQDYVRFIDNL
>lcl|NZ_CP026520.1_prot_WP_042226629.1_3584 [locus_tag=PC41400_RS18155] [protein=SDR family oxidoreductase] [protein_id=WP_042226629.1] [location=complement(4002424..4003326)] [gbkey=CDS]
MPEQDQYKVQDPTAQYPKPTPEWKQKQPEPGLQKKMTPVPDAGESSYRGTGRLTGRKAVVTGADSGIGRAAAIAFAREGA
DVLLSYLPEEEADAQEVVKLIEEAGRKAIAFPGDLKDEQYCEKLIATAVDKLGGIDILANVAGKQQFVENIADLTTEQFD
ATFKTNVYSLFWLCKAAIRHMKPGGSIINTSSIQAYKPSPILLDYATTKSAINTFSKALAQQVASKGIRVNVVAPGPVWT
PLQVAGGQPTEKLAEFGTNTPLGRPGQPAEMAPAFVFLASQESSYISGETLNANGGTPTP
>lcl|NZ_CP026520.1_prot_WP_042226631.1_3585 [locus_tag=PC41400_RS18160] [protein=isochorismatase family cysteine hydrolase] [protein_id=WP_042226631.1] [location=complement(4003623..4004102)] [gbkey=CDS]
MNIALLIVDMQIEFLQDRPVSTASAYINHTAQILRSGGHPVIHIQDVEEAGTISPERLEFIPEIQMEKDDLIVKKTFSNA
FWQTELEQILLDRQIGLVVIAGFDAGNCVLFTYNGAIERGFKAAILQHGILSAKEDLVALMYRDRHVISYPVIEFMVQS
>lcl|NZ_CP026520.1_prot_WP_042226634.1_3586 [locus_tag=PC41400_RS18170] [protein=ABC transporter permease] [protein_id=WP_042226634.1] [location=complement(4004501..4006417)] [gbkey=CDS]
MLTLFELVIRSMRKNSKPYALYFFALSSSATLYFVFAALQNDTSLLAATFGDIQFASLFEGAGIFLLAIMAIFMINANSI
FLRMRSREIGLYQLVGLTRREAGQLLVTEHVLLGIGALLIGMGAGTLGSRLFLLILMKLIGNESIIALTFSASAAIQTAL
AFAVLISLTVIQVLFKVYRTALLDLFKADRQGDRPKQPESALAPAMGLIGIGLITTGYVLSGSLPGRELVVQLTVVLALT
ILGTYLVFRVTFGWLLDRFRRSKNGQLGLTNSLSLAPIMHRMKGNASSLTLITVLSAMTLTMVAIAYSLYYSAESESRAM
LPYDFIFENNEQDAKLFRAELDKAGYAYVHQPVEAVRLTSLSGKEGEDGGSLLLLSAEQIQADGTDIIMPEAGEAVWYKG
QRRVLSKEAETAALPQTMVLGTSAVPLIIRVTQVVDRYTMNFSVKGRQLVVPETTLDAVREQTSASPEEEAIRLDTYRIA
DKSERAAASRLYAKYVKTDEYRPDFYTYFKESLHKFGLIIFAAGFLGLIFLIATGSILYFRQITEAEQERRSYTILRQLG
FGEREIMSGIIRKQLFMFAVPLAIGLMHSIFAVKAASALTLSDITFPAAIAMAVYTLVYFVFAVLTVTHYRRMVKIAE
>lcl|NZ_CP026520.1_prot_WP_241688690.1_3587 [locus_tag=PC41400_RS18175] [protein=ABC transporter ATP-binding protein] [protein_id=WP_241688690.1] [location=complement(4006401..4007186)] [gbkey=CDS]
MNPNHLQKTVLHARQVRKSFGARGHVQQVLRGIDLRVLEGEFIGIMGPSGSGKSTLINVLATLDKPTEGEIAVDNADISA
MKNTELSTFRRKKFGFIFQDYNLLDTLTVKENILLPVSLGKMKKEAAEAEYQAIAADLGILELAHKYPHEISGGQRQRTS
AARALIHKPSIVFADEPTGALDSKSASSLLEMMERLNRQRRITIMMVTHDPVASSYCSRVVFLKDGMLYSELYRGGKTRQ
AFFKDILNVQAVLGGDHVNLV
>lcl|NZ_CP026520.1_prot_WP_042226637.1_3588 [locus_tag=PC41400_RS18180] [protein=sensor histidine kinase] [protein_id=WP_042226637.1] [location=complement(4007268..4008290)] [gbkey=CDS]
MLLRYLADRKSWIVFYLLSLLFVDLLLWIDRGIAIQAVSMLYLNMLLLVSMAIFLGWRFRKEMKFGRAVAELAEKLEEDW
MEALPPAYFYHEKAANRLLRAAGHWYSHKLSDSSRARTVEEEYVAAWVHEVKAPLTAMKLTIDTLRSDPALRKIESEWLR
IHLLIDQQLYISRLPSLEADYLLEETGIQKLAALEVRELASWCVEKNIAVEFYGEEAYVRTDRKWCRFIIRQLLTNAVKY
SPAGGMILLSTTVLPDGPIRITIKDEGPGIAPHDLPRIFDKGFTGGNGRLLNAATGLGLYLARIVALKIGISLTVQSELQ
AGTAMEMTFPVSNSFESVRI
>lcl|NZ_CP026520.1_prot_WP_042226638.1_3589 [locus_tag=PC41400_RS18185] [protein=response regulator transcription factor] [protein_id=WP_042226638.1] [location=complement(4008303..4008986)] [gbkey=CDS]
MDINIFIIEDDAAIFRSLKERLEQWSFHVTGPERYDDVMNTFINVQPQLVIIDIQLPMYDGFHWCREIRAVSKVPILFLS
SRDHPLDMVMAMNMGADDYIQKPFHMDVLLAQVQAILRRTYAYAEEAADLIVWNGSVIDLKRGLIRKHGQDVILTRNEFY
ILVVLVKARDTVISRDDLIRKLWDDERFVNDNTLTANITRLRQKLTVFQLEEAIVTKKGFGYMAATM
>lcl|NZ_CP026520.1_prot_WP_053228633.1_3590 [locus_tag=PC41400_RS18190] [protein=serine hydrolase] [protein_id=WP_053228633.1] [location=complement(4009206..4011287)] [gbkey=CDS]
MNKQKNRSRRTFCILAALLAQSLFAESAVYAFHQPAPNSLASGKRPSSIGSPAPGDFTNGPHDGKEVEAFLDGFFANDAI
KQKAGAAVISVVHDGKVLVSKGYGVTDRMSQSQVDASRSTFRIASVSKVFTAAAVMQLVEQGKISLGDNIETYLDGYKIT
NPFDTPVTIENLLTHTTGFEVREASDADFLTNDSQKPVTLKESIFHQFPPVVRKPGMSYMYDNFASKLLGYIVQQASGEP
FGSYMQKNLFDPLGMTSSSFSLNNELSERLVKAYDPANSAIPVYDLSPREWPEGSMISTASDMALFMKAFLGGGRAPDGT
VILSPESVKGMSTYHTAIHRDVPDMTYGFESPVDPSRANGEDVISKGGDILGFSSLLWLLPDRKTGIFVSYNSNGDLRND
LFAAFMDHYYAARKSTFGPEGYKPQPQNALAKFEGLYSDLRIKLLTRVEAAGDGSLLVSDVSGRHRLKQVDDLLFVDEQG
KPLAFKKDADGRIAYLKYLNVFSYGAKIPENQEGYPDVPVGHPYATYILGLKSLGFLTDERSQPFQPQEAVSRGAFIHAF
NAIWSIPGSAHPSSFKDIETSPYRKDIQAAVEAGILNGTGNGLFEPDRPITREEAASIVFRLLVETGIRVPDSTARLAPG
TSEWAADAVSSAVIWKLHGPEVTESGGVIDYGSQRALNKQELAAMLFTMLLPK
>lcl|NZ_CP026520.1_prot_WP_129112509.1_3591 [locus_tag=PC41400_RS18195] [protein=catalase] [protein_id=WP_129112509.1] [location=complement(4011663..4013111)] [gbkey=CDS]
MEGNNGRFQSDAASSRSGSPQKPAAQHVPKEEPPRGRIPAQWAGNTPEVFHSQTIGERGLVLEQDNLLHETLETFVHTKM
IERPVHVKGSGAFGYFQTVHSMAAYTKLCFLQNPGQQVPVTVRFSLAVSNKGTPDTSRNVRGFSTKFYTEQGVFDLLCNH
IPVFSVRDAIRFPEFVQAFSPSPVNNLIDPERLWSFVARAPESTHFLVRLYSDAGTVKSYRHIPGHSVNTYVWRNARGVR
TYVKYHWIPFAGVQVIDRREAARLSENPDFAGKDLFDTLAAGKTVEYGLHVQLMNPDDEANLPFDPLDDTKVWDEKQYPL
LPVGRLVLNRNPDNYMDQVEKIAFSPSNLLDGAELSDDKLLQGRANIYWDSQRRRLGPDFRKIPVNHQQNWSPDSLVTSG
NGRYVEGRLIRSELPKPDNFTQAGEYYRALPPVQQEHLAENLSADLAGISHGTQSVVLSYLYQASAELGERVARQIQTQT
KG
>lcl|NZ_CP026520.1_prot_WP_042226640.1_3592 [locus_tag=PC41400_RS18200] [protein=SRPBCC domain-containing protein] [protein_id=WP_042226640.1] [location=complement(4013365..4013874)] [gbkey=CDS]
MTNPMTSKVEDKTLTLERVFKAPRELVFEAFTQAEHLKHWWGPRGWSIPVCNMDFRVGGVWHYCMKCEDKNQGDFYGMES
WGKAVYSEIQAPEKIVYTDYFSDAEGNIAETMPATVCTLTFIEVEGGTKVISKGEYATAEGLKQVLDMGMLQGITETWDR
LGEHLESIQ
>lcl|NZ_CP026520.1_prot_WP_084706532.1_3593 [locus_tag=PC41400_RS18205] [protein=metalloregulator ArsR/SmtB family transcription factor] [protein_id=WP_084706532.1] [location=complement(4013910..4014248)] [gbkey=CDS]
MDARVMSALAEPNRLLILDLLRECPLTVGEIAERLQMRQPQASKHLRVLSEAGLVKAQADANRRIYSLQPEPFRELDEWL
QSYRVIWEERFDRLDDYLQKLQGRESGEDRQD
>lcl|NZ_CP026520.1_prot_WP_042226642.1_3594 [locus_tag=PC41400_RS18210] [protein=Lrp/AsnC family transcriptional regulator] [protein_id=WP_042226642.1] [location=4014582..4015034] [gbkey=CDS]
MDSIDMKIIRLLEEDGRLSHEEISKCLHISRPAVHQRVAKLEKNGIIKGYRGIVDWSRLEQKIKAYIYVKMKCLNFRETS
DKIVALQVRDVTILECQRLAGEWCMMLKVRVSAPEQLTNLIDEMVKIHEVQETSTTFILSTMYENGRKEN
>lcl|NZ_CP026520.1_prot_WP_084706510.1_3595 [gene=yedA] [locus_tag=PC41400_RS18215] [protein=drug/metabolite exporter YedA] [protein_id=WP_084706510.1] [location=4015083..4016063] [gbkey=CDS]
MKQPPGTDLTQAGTSKFLGTAAALMSVYLFWGGTYLGMKVAIETIPPFTMGGVRFLSAGLILYVLARLTGEKRPNAEEWK
GAGITGALLLLGGNGVVAWAEQRVPSAIASLLVATVPLWMLVFGWMGRSRNKPSTGIVAGLLLGFAGIVILVVHPGSSSG
GNLDMLGIVSLLLASVSWSIGSLYSRKANLPESPLLSTALQMIIGGALLLITALVLGEWSGLQPSTITLRSYAALGYLIA
FGSIVSYTAYIWLLKHAEPVLVSTYAFVNPVVAVFLGWSLAGEQLSPNSLTAAVFIVAAVVVITLSRKGKGARKPPGRSA
SVTSRD
>lcl|NZ_CP026520.1_prot_WP_042226644.1_3596 [locus_tag=PC41400_RS18220] [protein=hypothetical protein] [protein_id=WP_042226644.1] [location=4016406..4016783] [gbkey=CDS]
MPFSTGVITNTRDFGTASSNIVMNVRNLDTTNPVTVTVQIFASVNSTTFYTAYLISYVVPANSYNVREFFIAGNTSYEVQ
INTTSVVTPNALLSVFGIDGFGNLVTDQRILQSELSFIPSLSPVM
>lcl|NZ_CP026520.1_prot_WP_241688691.1_3597 [locus_tag=PC41400_RS18225] [protein=DUF1835 domain-containing protein] [protein_id=WP_241688691.1] [location=complement(4016870..4017919)] [gbkey=CDS]
MGGKPGIARVFDRLGNQGTKLYLGFALEQIRKLKEREAPREKAISELIELYDELIESAEKRAAWDPVPTCTHVHFVVGDS
FAGSLKVALNQLNWSGTHKVITLSENYAIGPIRGLDLYEGWKARWDWFQDHITEACEVCDDPEEDYNQLLEQVNKIPEQA
KVIIWTSRNACEQTGMLHAIRLLSGRSNSLVLYDACAICEELDNRPDAAISYRHSGEIPSDKLQAALVRMKDNSCLRDAD
RTRLEREWQTISKQSGTLRIWRDGAVAEVPDNYYDMYLLGKLDKLRPASDHGYLKTARLIGEAMGYCEQFIGDAYFEYRV
RQLIYDGILEIKGVPSSMRRYSVGRKKIT
>lcl|NZ_CP026520.1_prot_WP_042226645.1_3598 [locus_tag=PC41400_RS18230] [protein=sulfite exporter TauE/SafE family protein] [protein_id=WP_042226645.1] [location=complement(4017982..4018737)] [gbkey=CDS]
MTISLIVLVVAAVFVGVFMKSMFGFGEAIVSMPLLALLPVPLQTSVALIGLVGLTVALLTVFSGWRHVERPVLIRLAAST
VIGIPFGLIMLHLVPSIVITSALGIFLIIFGCYSLLKKILSRKIGRPLLNNRGWVLPFGFASGVLGSAYNFNGVPVVVYG
TLRRWNPDRFRGTLQAHFLISGLLVVTGHALGGLWTLDAFVLYGYSLPAILAATGLGVLLNKRISADKFERYLFMMIILL
GFLLILPSGSA
>lcl|NZ_CP026520.1_prot_WP_042226647.1_3599 [locus_tag=PC41400_RS18235] [protein=YheC/YheD family protein] [protein_id=WP_042226647.1] [location=complement(4019095..4019835)] [gbkey=CDS]
MGSNTRNKWYKYLILRRHASLKRYVLKTRLLTRHNFSKLLSKYTHVIVKPVWGSRGRGVIQVSWKGNHKYAVRYENRKII
IRGRTNTYRYIRRKIGYASYMVQRRITRPTINKRPFDMRVIIQRRMYSSEWVVTGKVIKVAGKGYIVSNHTRSKGTLHRF
KSGIRRSSVRHLSASALESHIDRVSIRSAKRLSAFFPGHRIYGLDVGINRKGRIGIIEANLYPASSHFLKLKDRKMYYRI
MDYKNG
>lcl|NZ_CP026520.1_prot_WP_042226648.1_3600 [locus_tag=PC41400_RS18240] [protein=2-aminoethylphosphonate--pyruvate transaminase] [protein_id=WP_042226648.1] [location=complement(4019851..4020963)] [gbkey=CDS]
MNSSVKKNVLLTPGPATTTESVKRSQVVPDICPREAEFGQIMEYISTELTRLVADPDHYTTILFGGSGTAAVESIISSVP
NHDTVVIVNNGAYGKRMCEIAKVYGVNYFEFKSPPDDAIDLADLERHIQSSKRTISHLAVVHHETTTGLLNKIKDIGELC
KRYQIDMIVDAMSSFAAIPIQLKEMNIAYLAASSNKNLQGMAGVSFVIAEKNKLEHLKDQKPRSYYLNLYAQYNYFAEHG
QMRFTPPVQTLYALKQAIEELKQEGVDKRYERYATSWKTLINGLTRLGLTYIVPEEHHSKILTSILEPQCAGYNFQSMHD
YFYSNGMMIYPGKLEKLNTFRIANIGDITYKEMELFLELLERYLSSLKCG
>lcl|NZ_CP026520.1_prot_WP_042226650.1_3601 [gene=aepY] [locus_tag=PC41400_RS18245] [protein=phosphonopyruvate decarboxylase] [protein_id=WP_042226650.1] [location=complement(4020960..4022120)] [gbkey=CDS]
MDTKLLGEELKKLGYTFFSGVPCSFLKNLINYAINECDYVAAANEGDAVAIASGAYVGGKKSVVLMQNSGLTNAVSPLTS
LNYPFLIPLLGFVSLRGEPGIPDEPQHELMGQITTQMLDLMDVEWQYLSKDFEEAKKQLIQADEHIARNRPFFFVVKKGT
FDKVQLRNQQTSIHLNQIRQHAYADHQMPTRYDALSAINSVKDGNTVQLATTGKTGRELYEIEDADNNLYMVGSMGCISS
LGLGLALSQPSKDIVVIDGDGSLLMRMGSLATNAYYHPANMIHILLDNNAHESTGGQSTVSHSIDFIDIAASCGYTNSVH
VHSLEELQTFLQEWKVNKGLTFVLLKISKNSKDQLARPHMKPHEVKERLQRFIHNDPLKDKEVGGS
>lcl|NZ_CP026520.1_prot_WP_042226652.1_3602 [gene=aepX] [locus_tag=PC41400_RS18250] [protein=phosphoenolpyruvate mutase] [protein_id=WP_042226652.1] [location=complement(4022120..4023019)] [gbkey=CDS]
MKKTKQFRDLIHSQKPEFIMEAHNGLSAKIVEEAGFKGIWASGLSISAAMGVRDNNEASWTQVLDVLEFMSDATSIPILL
DGDTGYGNFNNARRLVKKLEQRHIAGVCIEDKQFPKTNSFINGQEQSLADIEEFSGKIKAMKDTQTDEDFTVVARVEAFI
TGKGLKVALERAEAYRKAGADAVFIHSKRSDVAEIEAFMKEWAGRLPVVIVPTKYYTTPTVRFQEMGISLVIWANHNLRA
SIQAMKHLSDRIYRDQSLVHVEKDVAPLEEVFRLQGAEELEAAEQQYLPSSEPDSGEHA
>lcl|NZ_CP026520.1_prot_WP_042226654.1_3603 [locus_tag=PC41400_RS18255] [protein=YheC/YheD family protein] [protein_id=WP_042226654.1] [location=complement(4023062..4023790)] [gbkey=CDS]
MSVGKMKKYKVMLKHPVLRRHLPKTRWFTKSRTKRMLKTYSSVFIKPNHGSGGSGIIRVKRLKKRYEVRSASRRKVVRSR
FLFKAIKSYRNPGHQYLVQKGLRLAKYHGSIFDIRIYLQKPRGKWVVSGEVARVAAPRKYVTNYLKGGHAVSLNRVLYSL
FKHNRTKANTYSRKISKLCVIIAKTINKRHPVRELGVDIAIDKKGRIWIIEANSRPGHMLFTQLSDRTMIKTIRRNKRLI
RK
>lcl|NZ_CP026520.1_prot_WP_042226657.1_3604 [locus_tag=PC41400_RS18260] [protein=NAD(P)/FAD-dependent oxidoreductase] [protein_id=WP_042226657.1] [location=complement(4024136..4025188)] [gbkey=CDS]
MNDVEVIIIGGGQAGLAMGYFLKQKGVTFLILDAHKRTGDSWRARYDSLTLFTPRMYSSLPGMNFPGEPDGLPSKDETAD
YLEAYAKAFALPIRHQTKVHSLSKHEGGFKIETGQETFIARQVVVATGPFQEPVIPDFAKKLPKEIVQLHSSGYKNESQL
ADGPVLVVGGGNSGAQIAVELSGRRQVTISVSRPMSFKPMQLFGRNIFWYFERFGLLKANAESLLGSWLKKLPEQVYGTE
LKSLLKKRIVSLKQRAIDADSAGVLFKDETRMKVSTVIWATGFRSAYSWSDIPETADVSGKPIHRGGVSKVKGLYYLGLP
WQTCRGSALIGWMGRDAEHLADVIAQHAKR
>lcl|NZ_CP026520.1_prot_WP_042226658.1_3605 [locus_tag=PC41400_RS18265] [protein=TetR/AcrR family transcriptional regulator] [protein_id=WP_042226658.1] [location=complement(4025225..4025815)] [gbkey=CDS]
MEQSAQETRDKILTAAISLMKVKGYKGMTTRAIAQAAGVNESTIFRHFVNKQGILDALVDRYSYVPGIHQVFDSLTGELQ
SDLIRVAEAYQEFMKHNGEMVLIGLRETGQFPELDEKTAAIPIQLKKLLVDYLTRMHEKGVIRRTDFEASAMTFIWMNLG
YFISASLYGQKITIVKVEDFLQYGVRTFAQGLSEEP
>lcl|NZ_CP026520.1_prot_WP_042226659.1_3606 [locus_tag=PC41400_RS18270] [protein=HAD family phosphatase] [protein_id=WP_042226659.1] [location=4026049..4026756] [gbkey=CDS]
MEDLIKSARKAASIEAVLFDMDGVLIDSEPIYFDIERSSFNHFGVPVSEEDHHSFVGVTLESMWEQILDKHRIPFALEQV
LTYHKDNVMNILSGHTELTAMPQVERWLSWLKEKQIPIAVASSSPRPLIELIMDKTGLGSYFDVWVSGEEVKHGKPAPDI
FLHAAGLLGVEPSSCLVIEDSRNGVKAAKSAGMRCVGIQNPGSGNQDLSLADHRVRNFEELWALKDSLPFAPFSS
>lcl|NZ_CP026520.1_prot_WP_042226661.1_3607 [locus_tag=PC41400_RS18275] [protein=VanZ family protein] [protein_id=WP_042226661.1] [location=complement(4026814..4027734)] [gbkey=CDS]
METKLRKLILTGIVLYTLLILYFMFFAFNRVEHEYSEYGYVFMLLPEAVPLLFPDLSDLSFSWIYNLGNIAAFIPYGIFV
PLLHRTHVWKFLSWFVMAIIGLETLQSLTFLGTFDVDDIISNTLGASIGWCVYKVAASPKIFWKKLMASGLTVAVLLIGI
MSLSELIDFTLQKKEGLLLALNEMKEAADNRPLTKDLPEFTVGGEKIKPNLNVYGSKGKESQTYTYNWAAKKEAHLYAYY
GIPDNGDSAGELIITADGQERVYYTESIQAVPVDIPFDQLNELTITLKGNAKLWDVTISERKHWWE
>lcl|NZ_CP026520.1_prot_WP_042226663.1_3608 [locus_tag=PC41400_RS18280] [protein=hypothetical protein] [protein_id=WP_042226663.1] [location=complement(4027966..4028277)] [gbkey=CDS]
MSQRYRITRAFQENGNSAKTISLEECKQIFSSKADFTYTSVFTVKGSTSMSIEGDFFIWSYDEDTQIPFRHYDGDLYVSG
TNKVVIPVVIEVASELEADVQEE
>lcl|NZ_CP026520.1_prot_WP_042226665.1_3609 [locus_tag=PC41400_RS18285] [protein=TetR/AcrR family transcriptional regulator] [protein_id=WP_042226665.1] [location=complement(4028350..4028964)] [gbkey=CDS]
MVNQQDPRVLRTRQLIREAFRDLLQRKEFDAITIKDIAQKATINRATFYAHFEDKYALLDEITELAFHEMIPEQVADAQE
FTDEICNQLILMTHRYIVDFYRICRMDSYPMATLVDEKIKKMLQQAIENIFLKGDTGHGADRHRIKIMAAMTGSAIYGAA
HHWLNAREKDRADVLVDIVRPYVMNGLGLYRNGDEVLNDSPTAP
>lcl|NZ_CP026520.1_prot_WP_042226667.1_3610 [locus_tag=PC41400_RS18290] [protein=NADP-dependent oxidoreductase] [protein_id=WP_042226667.1] [location=4029165..4030157] [gbkey=CDS]
MKAIALTSFGIPEVLEELEVPVPTITDTQVLIEMRVSSINPADILLRSGAILKSPMADKFPAQLPLVLGNEVAGIVKEVG
EKVRHFKPGDRVMGMIPMGSYMDYVAVEEDHLAIIPESLSFEEAGAAPAVALTAWQALFEHGRLQPGQRILVQAGAGGVG
HAAVQLAKQHGAYVIATARDYNHDFVKGLGADEVFDYTKVDFAAQITEPVDLVLDSAMDPSTFGSTFGTGLPGEIGKKNY
SVIKDGGTYISLAAFAINQYPKVRDIDAYFFQARPNRTDFESIVRHMKENKLNIHIDETYPFTAQGLLEAYRKSEEMPKR
GKIVLSKNTG
>lcl|NZ_CP026520.1_prot_WP_042226669.1_3611 [locus_tag=PC41400_RS18295] [protein=MFS transporter] [protein_id=WP_042226669.1] [location=complement(4030339..4031568)] [gbkey=CDS]
MATFFLLIIYMAFISLGLPDSLLGSAWPVMQTDLGVPLETAGLLFMTVTAGTIVSSMASGTLLKWFGTGKLTLISCIMTA
GALLGFAVAPSLVWLMLCAIPLGLGGGAVDAGLNNYVAAHYKAHHMSWLHCFWGVGATLGPIIMAQYISGPDSWRDGYLT
VAGIQFALVILLFVTLPLWNRVAKNNHTALGEEKEDLHGAEDQRSESSVKPLQVRGVKLALVSFLFYSGAEATIGLWGSS
FLVNMKEVPAAVGAQWVSLYYAGITVGRMITGFITFKVANRTLIRAGQITALAGAALLILPLPPVFSLAGFMITGLGLAP
IYPCMLHETPVRFGRKHSQSIMGYQMAVAYVGSTFMPPLLGFLASLFGIGVFPFYIVLSAAAMLLFSEKLNAFLTARKNA
LKNNTTFTG
>lcl|NZ_CP026520.1_prot_WP_042227420.1_3612 [locus_tag=PC41400_RS18300] [protein=ROK family protein] [protein_id=WP_042227420.1] [location=complement(4031572..4032567)] [gbkey=CDS]
MKKTILSSIRAALLELGSATKVELSHKLGISFPTVSKFLTQMEKDGEIFSAGLDGSSGGRRAKRYVYNPDYLLGLALFLE
KAETNYTIFNASGEIKEQGTTSGVLLDPVICLSEQIGKILAKYPKIGSIAVGVPGAVNNGRIIYIPGYEQFHHFDLKSDF
EERFSVPVAVENDMNAAILGYSLNAAKTENPSMIYLYFGQNGPGAGILVNGDVVRGSTFFSGEVSFVPQYDDRNFLQALK
REAGPEEIADCEKEIDAISRLVASFTAILNPHAIIFCDDEANRQLLDRIASRSAAYIPEEHLPELTTSDWKRDYLRGLQR
LGLDLMIREKR
>lcl|NZ_CP026520.1_prot_WP_042226671.1_3613 [locus_tag=PC41400_RS18305] [protein=ABC transporter substrate-binding protein] [protein_id=WP_042226671.1] [location=4032877..4033833] [gbkey=CDS]
MKRHHGMPIMLWGGIFLLLLIVSACSNAGGHASDVQLEKNEKTFQYTTITGEQIEIPAQPKQVVYIGPTLGDFLVMDIPI
VGSNLIHTTHSYYKGKTEGIVDVGNPGDLEVILSLKPDLIINSYYNADAEKNAAFSKIAPTIPFNPALPYAERVKEMGKI
FGQQEKAAQLIAKFETQSKEMFDKLQLADGETATVFYQLGKTLYVMGNRSLGAVIYGKNGFAIPPAVQKNIIDQKGVSFV
KVSEELLSEFAGDHLFVLVQDNSESKTESDQLLKSPIWGTLPAVQKGNVYVVSAEWNTDGLLALEQLFKELPQWMQRK
>lcl|NZ_CP026520.1_prot_WP_042226673.1_3614 [locus_tag=PC41400_RS18310] [protein=AraC family transcriptional regulator] [protein_id=WP_042226673.1] [location=4033969..4035633] [gbkey=CDS]
MNPAPLYIRLQHYDRFQVDRLMPASSHNEAIDYKGTNLCTLLIALSHGIRLYIDKIAVDLRQGSCILVSPVQHYSVESSN
GEAQLARFIFETFEVEGLILNPVAHPPLLCGYPYSLFFSQVKQVLGNESWMRNRFCSSLSTSEMAMMQGRLQLILGLMAQ
LDQQAAHLQSEEKIKMIQYTVHYMEQHYDEDLTVEQLANMAGMVRWQYSQQFKTLTGQKPTDYLVHLRIKHAKKLLCNST
EPLSKISRQIGFKDEYYFSRCFRKLTGNTPREYANIHLHTQLRTVIDSLGRKVHVPKNATRIVTDGKYTLGELLVLGISP
IGAAISMKNNVIYHNKLRNIQNIGHWADPDKIAPLQPELVLLSYHHHAQDLQVLDAIAPTVVLDHKFRLFDRLRYIAKLF
ERAKAAEKWIATYEDKVRRVRRQLADAYAAGETAAVYLKLGAKLYVMGHNGFAATLYESLGFRPSAKVMHLIEQGQSWIE
IQQHQLNHYAGDRNFFLASPLELQTAAHCPQIEAIAELTPGKVHYMDATWNYEDPITRGRLLEMLPYIFKRKTM
>lcl|NZ_CP026520.1_prot_WP_042226675.1_3615 [locus_tag=PC41400_RS18315] [protein=NAD(P)/FAD-dependent oxidoreductase] [protein_id=WP_042226675.1] [location=4035728..4036789] [gbkey=CDS]
MTQALELYDMTIIGGGPAGLYAAFYSGMRDLKTKLIEAKEELGGRLLIYPEKMIWDVGGVTPIRCEHLIRQMIQQAHTFD
PTFVLGQQISGLERLEDGTMLLTAVTGEQHRTRTVVLALGHGVLKMAKLDIEGAERYEVTNLHYTVQELESFHGKRVLIS
GGGDSAVDWANALEPIAASVTVVHRRERFGGHEKNVKRMKESSVDVRTPYVLDRLHGNVSGDAIEAVSIARVNAEGLIME
ETEQLVIDAVIVNHGLRGDFGSIVDWGMDMGQWDINVNEKLATNLPGIFVAGDTANYPSKVNLIAGAFTDAVLAVNSAKL
YLEPAAAKVAYVSSHNAKFKEKNKALGMAEEDA
>lcl|NZ_CP026520.1_prot_WP_042226677.1_3616 [locus_tag=PC41400_RS18320] [protein=putative protein N(5)-glutamine methyltransferase] [protein_id=WP_042226677.1] [location=complement(4037131..4037886)] [gbkey=CDS]
MRVAAALREAGCVFAEEESRRLISAAQTPSELAAMVDRRSAGLPLEHVIGWAEFCGLRIAVDHGVFVPRQRTEFLVRQVA
ALAGKNAVVVDLCCGTGAVGVALASALQRIELYSVDIDPAAVRCARRNVSAAGGRVYEGDLYDPLPADLRGRVHVLVANA
PYVPTGSVRLLPAEARIHEARAALDGGEDGLDIQRRVAAGAPVWLAPGGHLAVETSRRQAPQTAEIFARSGLNARVVCSD
ELDVAVVIGSR
>lcl|NZ_CP026520.1_prot_WP_042226679.1_3617 [locus_tag=PC41400_RS18325] [protein=helix-turn-helix domain-containing protein] [protein_id=WP_042226679.1] [location=complement(4038134..4038802)] [gbkey=CDS]
MEALDPPKESTIGELIKIQLKKQSLSMRKLSSLSGMDTATISRIVNGKQEPKLIHLKLFSEYLDLSLDRLIEASGYIVNP
VKETGYSHIHQSLGVLLELFGDSSVSDLKATTGRIEQELAKYEVYAQTEEGHGLICREFEAKLAQVNGVGPLIDQLKEMY
TLYCAKSASETDQAILGSALLYFILSADIIPDYMFPIGYVDDAMAVYLVLDRLSQRGFPHKL
>lcl|NZ_CP026520.1_prot_WP_042226681.1_3618 [gene=ytaF] [locus_tag=PC41400_RS18330] [protein=sporulation membrane protein YtaF] [protein_id=WP_042226681.1] [location=4038988..4039617] [gbkey=CDS]
MAWLLILGFAISSSLDNLGVGISYGIRKIRIGMLSNLIVAGVCFLFSYTGIWFGKWISSVLPGMLPVLLSAFSLFFIGIR
IILMAAPGKEPEVAPSPSKRNGNGLRSILRNPEQVDRDNSKHIGIAESFILGIALSANALTNGLSAGLLGLSPFAISLTA
AFGSFLTVWLGVWLGGKVAGIRIGSFTLGQFGTLVSGMMLLVIAANSLL
>lcl|NZ_CP026520.1_prot_WP_042226683.1_3619 [locus_tag=PC41400_RS18335] [protein=DUF3995 domain-containing protein] [protein_id=WP_042226683.1] [location=complement(4040048..4040470)] [gbkey=CDS]
MKSVMVLSAVSILALISILHIYWAYGGRWGIGAAIPSKAGEYKPAFTPGKPATLLVAVLLLIVCVILLMQGSYIDPLPAN
ALTRTGCIVCASVFILRAVGDFRHVGFFKKVKHTVFARNDTWLYSPLCLYLGLTCAILLF
>lcl|NZ_CP026520.1_prot_WP_042226686.1_3620 [locus_tag=PC41400_RS18345] [protein=TetR/AcrR family transcriptional regulator] [protein_id=WP_042226686.1] [location=complement(4040949..4041464)] [gbkey=CDS]
MQVTREDWVKAGLNKLAEAGIHEVRVEALAKALKISKGSFYHYFRDRKELLDSMLDYWEEYATKRIIQNVEQDSFSLEQL
LNLSINNRDKKMEIGIYAWAKHDPVVASRLVDIEEQRIHFVTVLYEKRGLTFSEAADRARLTYLTYVGWMTRFESNTNFD
IVKMFEILLKS
>lcl|NZ_CP026520.1_prot_WP_042226687.1_3621 [locus_tag=PC41400_RS18350] [protein=aminoglycoside phosphotransferase family protein] [protein_id=WP_042226687.1] [location=complement(4041508..4042506)] [gbkey=CDS]
METESLKLARLTKLVTEHYHLEPLSINRQQGGWASLAYRIETPSNSTYFLKEYEKKRSSTPKLTARLHKYIPVTCWLNEN
TNLAGKIPVPLKTQSGEYVCENVDGIFLLYEYIHGETIGEKALSQTQIEQLTTILAELHKAGNSISVPAEHLTEDFHVPF
MNALNETMAARDLSPRLRSAIKPYKNVLFEHISVLQDLSLKLAGRMLPLRWCHTDIHPWNMMQSDHQLILIDWEGLKLAP
VEADLMFLTDETYFHDFFALYKKIHPDYIIDEEALRFYKIRRLLEDIWEFTEQFLYDEQSEPEKEGTLKQLIHELESLKC
GTQHSIDKIKPE
>lcl|NZ_CP026520.1_prot_WP_042226691.1_3622 [locus_tag=PC41400_RS18355] [protein=multidrug effflux MFS transporter] [protein_id=WP_042226691.1] [location=4042677..4043864] [gbkey=CDS]
MLTKKISTPSILLFIILVGFPQISETIYTPSLPDIADHFHVSQNLIQLTLSVYFLGFALGVFCWGRLSDVIGRRPALLWG
ILVYGLGSLGCYLSGTAEWLLVCRFLQAFGASTGSVVTQTILRESIDGAKRHAVFAQISAALAFTPAIGPLIGGWVDQAL
GFRAVFLTLVIMGAAVFAYTFAALPETKGAVTTTIKIRSVFRRMIGDKKLWAFGFLIGATNGLLFSYYAESPFIFIEYFH
LTPGEFGLFGILVALSSILGALLSKKLLTKCTAEKIIVTGTVLSAFGAACMTALILYKASPSAIFLILLATCIFIFLLGI
GMAIPNCLSLALVRYGDVLGTAGAIFGLGYYILVSLITSGMSYLHNDSLLTMPLYFLVLSLMMVFVSKRLPHSVK
>lcl|NZ_CP026520.1_prot_WP_042226694.1_3623 [locus_tag=PC41400_RS18360] [protein=polyprenyl synthetase family protein] [protein_id=WP_042226694.1] [location=complement(4043971..4046355)] [gbkey=CDS]
MNQTFTDQAETWCRLAEQRAAQYFASLYEQVKVKSYVPVLTEDIRVWKKDHIPPFSLVSFLSGRKRKPDSRDYSRYIQWL
RYTGKLDDYLDRSISYIYLRDLGQALDSPATQKKVHKLIGDIKNQLMDSAKADGGDQPDYISWAGLYRWAQKEGIETAAI
WLIAKLKNVSTRIPKEMNAEQAQRKLLKIILGVILHVDDDMDDKTPPDERSRRLDEAIRLGYSYGLTYPFIDDLLDSGIL
TDLEKEQFTGMIRNALLTGTVPELGEWSENNRILLEFVHAELRDAFQSIQAYQRPEALSTFYEQSYVFFHSQELDRIKDL
SCESYTNEELYIPIILKSSSSRLIVRSVLSADADEGFDNRTFYYGIYNQLADDFADMFDDRKDGAVTPYTYYLKYRDRRP
DLINPYELYWTVISHLIHNVYRSDAKTREVILDRAINGLKRCKERLGAEKYNELMNIFASGDPEFNRLIQTMVSKADDVD
FFDKLLRDQVVAVLQNNQKEKEAFSATVASVRDQINSFLHISKPHGTPGMGEALIDAANYSLEGSGKRIRPILTWVMGVN
EYGLKAKEIIPLLKSLEYMHTASLIFDDLPSQDNASTRRGRPTLHQVHNIATAELTGLYLIQKATQEQTSLDCFPAKSVL
ALIRYSAEKTEEMCMGQAMDLQARGKALTLEQLNRICFYKTGIAFEAALVMPAILAEVPAAEIDALKKFAYHAGIAFQIK
DDLLDAEGNVEVLGKPAGKDIENNNSTFVTILGQDGAGREMWEHYCLAMESLKEVPRNTAFLKHLLTYIVGRDH
>lcl|NZ_CP026520.1_prot_WP_084706512.1_3624 [locus_tag=PC41400_RS18365] [protein=ABC transporter ATP-binding protein] [protein_id=WP_084706512.1] [location=complement(4046597..4048324)] [gbkey=CDS]
MKMIKRFVTYYKPYKKVLIADLIFAALASMCALAFPMLVHFMTTSAISEEGIAAALFVKMIAVFLFLMAVEYGSGFYTDY
YGHTMGAAMEADMRNDLFRHVQRLPFSFHDKTATGQLMSRATHDLFNISELAHHGPEDLVISFVRIAGSAIILWQLNPVL
ALTLLSVLPFMFLFAYRYSIKVKEALKKNNERIGDLNAQLEDSLAGVRVVQSFANEEVETAKFRAGNEVFLGSRKKGYKA
EALFFNGLTSFMSLMYIAVVVAGALLISNNMLLLTELITFLLYINTITDPVKRLVNFTQNLQNGAAGFERFVEILSIKPA
ITDSEDAVTLNRVRGKIRFENVGFRYSGDNSPVFSHLNLTVSPGETVALVGSSGAGKTTLCQLIPRFYDPSEGNVYVDDL
NIKDVELKSLRSQIGIVQQDVYLFAGTVRDNIRYGKPDATDEEIIAAAKLAHAHEFITGLPGAYEAFVGQRGNKLSGGQK
QRISIARVILRNPAILIFDEATSALDNESERIIQHSMEMLTRNRTALIIAHRLSTIRDADRIVVLTEQGFAEEGTHETLM
RQGGTYAKLYNMQSN
>lcl|NZ_CP026520.1_prot_PC41400_RS18370_3625 [locus_tag=PC41400_RS18370] [protein=DinB family protein] [pseudo=true] [partial=3'] [location=complement(<4048576..4048734)] [gbkey=CDS]
MQALIEEYSRGYKLLREAVEGLTDKEFRFKPALDKWSIHQILIHIADSELVAT
>lcl|NZ_CP026520.1_prot_WP_042226696.1_3626 [locus_tag=PC41400_RS18375] [protein=hypothetical protein] [protein_id=WP_042226696.1] [location=complement(4048810..4049214)] [gbkey=CDS]
MDTNLHIRISKDLKESFQQVAKQNNKDTSSLVRDWISAYVAEYRNSDEETATELYVAGYQLQEALGGRDKVSKDFAKQLQ
EYSLTDQKEFTQLILTTYLDHDLTIPPVISKTYKGYAYSQMFLLGLIGDKPKER
>lcl|NZ_CP026520.1_prot_WP_241688692.1_3627 [locus_tag=PC41400_RS18380] [protein=MBL fold metallo-hydrolase] [protein_id=WP_241688692.1] [location=4049456..4050160] [gbkey=CDS]
MLELQLGPMTIHPVVVFDQESWTLIDTGMPGSAPAIMELAKQAGTGDLPLRSILLTHQDIDHIGGLPGILAATGGSAPVL
AHPGDEGAINGTQPMLKMTPERLAGFLGQVPAEVREQFERTFLNPSRPNVDRTFSDGETLPYGGGLTVIHTPGHTPGHVS
LYHAPSKTLIAGDAMIAKNGELTGPNPVNTPDMGRALESLKKFAAFDIERAVCFHGGLVQGGVNESIARVARQA
>lcl|NZ_CP026520.1_prot_WP_053228635.1_3628 [locus_tag=PC41400_RS18385] [protein=ABC transporter substrate-binding protein] [protein_id=WP_053228635.1] [location=4050593..4051561] [gbkey=CDS]
MKKIKYILLVTSLAAILLLLAACSSNEIEQQSTPNTETKTFKHESGETVVPVHPQRVASDQYMGQLLKLGIKPIGVRDNM
LTEPWIKPANLEKGALDGIKSLGGFPLNLEALTSLNPDVIISSLPDNIESYSKIAPTVHIPYASPNYSSPMDKFFKIAEL
FDKTEQAKQWIDQFEQEAARVQEQILQSGKIKKGETVSVMGVFNKNVYVFGPKGSYGGSMIYDYVKLQPTPQAEQLRGQT
FHSAQLSLEMVPEYMGDHAFFAVIDEAKGKELMESSFWKTLPAVANNQVYFYDSPIFAMDDPYSMEAQLKIIHDLFMPTE
KQ
>lcl|NZ_CP026520.1_prot_WP_042226700.1_3629 [locus_tag=PC41400_RS18390] [protein=AraC family transcriptional regulator] [protein_id=WP_042226700.1] [location=4051658..4053271] [gbkey=CDS]
MRNITPSSLAFHLLSLERIEANEKAKQFSINSFGLIAVVQGSLLSEDRNQKLTMTAGEVVVFNPTTEMFVHGENCTYYFI
SYQVFQPNAKEMSNPFFHFNSKEHSPAKVKIPIRSLTRGLEQMYRHAWSTDVLQQCTNHMYFQEWMTLILTEQLKQTNQE
HNSVQAVEQIIAYIHAFYYEELTVEQLAIKAGTSRRQFTHVFQKLTNRSVTQYVTNIRIEEAKKLLVQNETLASIAKVVG
YQDEYYFSRRFKQVVGKSPRQYAKLKKDGVKAVKLPVIPQRIVADQYMGQLLCLGIAPIGARTDMLNKDYIAPLQSTSTL
LHQTIDLGKDFFISPSRIVQLQPDLIITQNEANYKTFGSIAPTVVIPYPHTSPLTKLRLIGEAVGKLKQAEQWIARYEYR
VERIRSQMISRLGTSLSVTNLLLLTNKLYVLGPYAGYGSYSLYQALQLQSPAKQHKELTELSLSAEIPISNLPEYVGDHI
FLSAYGDTSILTENEAWKELSTVQKGNVTFCNPYRFAFEDPYSLDEQLTTIKRALLI
>lcl|NZ_CP026520.1_prot_WP_042226702.1_3630 [locus_tag=PC41400_RS18395] [protein=DUF1801 domain-containing protein] [protein_id=WP_042226702.1] [location=complement(4053346..4053696)] [gbkey=CDS]
MNPTNQEVTDFIEALKEPWQKELSAGLREVIHQAIPEVSERIQYKKPHFLKNGKYAAVISTSKDAVSFTIFNASDLQLPE
GMFDGPPERKTLKLRKGETFDSGLLVPLVVQASSAL
>lcl|NZ_CP026520.1_prot_WP_042226704.1_3631 [locus_tag=PC41400_RS18400] [protein=YciI family protein] [protein_id=WP_042226704.1] [location=complement(4053693..4054145)] [gbkey=CDS]
MRFMLILKATGYSEAGVKPGPAHSEAMAAYRKSLAEAGALLAAEELQPSSAGIRIQYFPGEEQPVFEAGPFVTGQGLLAG
YTLIEVNSEEEAVNWALRMPVPQGFGTFGIEVRKLEEQADYARNPINRALVADLEDQLELLKTYLKETKI
>lcl|NZ_CP026520.1_prot_WP_042226706.1_3632 [locus_tag=PC41400_RS18405] [protein=RNA polymerase sigma factor] [protein_id=WP_042226706.1] [location=complement(4054254..4055519)] [gbkey=CDS]
MKQSSAHRTIDAIWRIESPKLIAGLTRMVRDVGLAEDLAQDALVIALEKWPISGIPDNPGAWLMTTAKRRAIDLLRRTKV
RDRKYAEMAQNMDLYTEEDLNATLDGEIGDDLLRLIFMTCHPVLSQEARVALTLRLLCGLTTDEIARSFLAAEPTVAQRI
VRAKKTLSAAKVPFEVPAGEALKERLTAVLEVMYLMFNEGYSATSGEDWIRPLLCREALRLGRVLAEIAPGEGEVHGLVA
LMEIQSSRFRTRIGAAGEPVLLMDQNRAQWDRLLIRRGLAALERSRKLGRPLGPYALQAAIAACHAEAPTAAETDWIRIA
ALYEALSRAAPSPIVELNRAVAVAMAFGPAFGLQIADELSGEPALKGYHLLPSVRGDLLVKLGRLEEARAEFERAASMTR
NARERELLLNRAAECGTEAAE
>lcl|NZ_CP026520.1_prot_WP_042226708.1_3633 [locus_tag=PC41400_RS18410] [protein=YciI family protein] [protein_id=WP_042226708.1] [location=complement(4055516..4055950)] [gbkey=CDS]
MRFMMIVKATTDSEAGVMPSQELIEAMQKYNEELVKAGVLLAADGLQPSSGGIRISYPEPGGKPKVVDGPFTEAKEIIAG
YTLIEVKTREEAIEWAMRMPDPHGYGQGEIELRQVFEVEDLMSNPETVEKEKSLRRQVEEQKKA
>lcl|NZ_CP026520.1_prot_WP_241688693.1_3634 [locus_tag=PC41400_RS18415] [protein=alpha/beta hydrolase] [protein_id=WP_241688693.1] [location=complement(4056140..4056979)] [gbkey=CDS]
MMNTVTSKDGTTIVYDKTGQGPALILVTGAFSYRKFPGQVQLAELLSAQFTVYNYDRRGRGDSGDKQPYAIEREIEDLQA
VIDEAGGSAFVWGLSSGAVLALQAAANGADIAKLALHEPPFVVDPADRKPPHNFTAHVSELIAAGRRAEAIKYFMTQGMG
APSFVVGMMKLMPGVWSNLMDVAHTLPYDAALLDGYMEGKPLPAGLWRSVTQPVLVLEGTESPAGLRHGAQALAQALPNA
QLLSKKGLGHTKKLPAKLISAELSAFFTGNSEVRIAANR
>lcl|NZ_CP026520.1_prot_WP_042226711.1_3635 [locus_tag=PC41400_RS18420] [protein=MFS transporter] [protein_id=WP_042226711.1] [location=complement(4057329..4058597)] [gbkey=CDS]
MNWLKWDLNLKIRLTGDTVFNLLFWMYFPFITLFFSEAFGKSVASVLMAVPPIFSLAGGLFGGFLSDRLGRRPAMLLGAF
MQASMFALFALSSSHWLDYFAYIGIGLGGAIYMPASSAMVADLTPVKDRRMVYATFVTGRNIGAVFGPALGAVFFFQYRH
ELLWTCTAVTLLYSLAILLMIKETLPAAAVAARQGMRLSGILKEQVSSYGVIFRDKLFALYIAAGILVTIAFMQLDMYLA
VYVTEYVPAQKLLTIGDWSFSLSSSQAFGLILGLNGLLFVLGVMPVTKWFEEWPERNVLILSSLLFGAGMFLFGFTTQLW
LLLIFTVIFTIGEISRSPVTESFISKYAPDHARGQYMGASGLQFSVGRLFAPVTVLLSTWFPPLVVFGFIFLVTILSSLL
YIRVFKMLPASYYDPQREADAV
>lcl|NZ_CP026520.1_prot_WP_042226713.1_3636 [locus_tag=PC41400_RS18425] [protein=ABC transporter ATP-binding protein] [protein_id=WP_042226713.1] [location=complement(4058758..4060779)] [gbkey=CDS]
MNVGKKLFKYVLYYKKSFIGALCLLLLMVAAELSGPLIAKQLIDRHILGIEKPWYETREPGENTVTYKGKYYKRGDYFNE
QEEKGQEVRILQVQRQFVFVPEAVKVDGERLVQGEQLIIKRNAETGTYPAEILSTRDLFLFYEPELGKIVNLILLYFGFV
AAASLLEYGQTYALQVSSNRVIQKMRIDVLKQIHRIPIPYFDRIPAGRIVSFVTNDTEAVKELFVTVLRNFFTGAVYITG
ILTALFILDVRLALICTLIIPALAGWIYLYQKFASKYSQTIRSTLSDINGLINEAIQCMPVIRAFSRQRTVRQEFEDKNV
RYFTYSNKLLTLKALTSNNLVTVLRSLALTAIIWYFGGAALSGQSVFTFGMIFAFVEYVERLFQPLVGMVNQLAQLNHSL
VSAERVFRLMEEEGSDVKDEETPRLKGNVKFEHVSFAYSDDHYVLRDISFEAKQGQTIAFVGHTGSGKSSIMNVLLRFYE
LQKGTVTIDGHNIQTLSRQQVRKQIGIVLQEPFLFTGTIASNISLDNPAISRELIEDSLRAVGAERVLQHLPKGLDEPVI
EKGSTLSVGQKQLISFARALAYNTPVLILDEATSNIDTETEGMIQSALEVLKKGRTTFVIAHRLSTIHHADLILVMDRGQ
IAERGNHDELMRQRGIYYQMYQLQMRDRANVPV
>lcl|NZ_CP026520.1_prot_WP_042227427.1_3637 [locus_tag=PC41400_RS18430] [protein=ABC transporter transmembrane domain-containing protein] [protein_id=WP_042227427.1] [location=complement(4060794..4062542)] [gbkey=CDS]
MVSVLVRLGWFFKEHKIRYTLAVLILILVNGVEMLPPLLLGSALDHIQQNTLTPASLAQYIGFILGLAVVIYTLNYAWMF
QLHGGANRLQLKMRSRMMAHLLKMTPSFFEKYRTGDLMARATNDIKSIGETSGAGILTLVDSCLFSVTILITMGFFIDWN
LMLVTILPLPLIAVVITFFGKYLHRRFTIAQDAFGALNDQVLETITGIRVVRAYVREEANQKNFETMAEDLFRKNVAIAR
VEALFEPTIRMLIGISYLIGLGYGTYLVFQQRISLGELITFNVYLGMLIWPMYAIGELINIMQRGSASLDRLDEIMTYSP
DVRDGEGLEEEIALPHTIQFHQVSFRYPSSAVDNLSDITLHINRGQTVGIVGRTGSGKTTLLRQLLREYPADRGEITVSG
IPLERIRLETWNKWIGYVPQEQMLFSKTVKENVQFGKENSTEQEVFRALELASFMQDIHTLPDGTDTLVGERGVSLSGGQ
KQRIAFARAIISEPEIMLLDDALSAVDAKTEASIIANLIAERKGRTTLISTHRLSAVEHSDFIIVMDQGRIVQCGTHVQL
IACHGWYKEQYESQKLQMNLLK
>lcl|NZ_CP026520.1_prot_WP_042226716.1_3638 [locus_tag=PC41400_RS18435] [protein=GMC family oxidoreductase] [protein_id=WP_042226716.1] [location=complement(4062597..4064111)] [gbkey=CDS]
MTKQFDVCVIGSGASGSVVSKYLAEKGLEVVIVEQGELIEPGTNIDSVMSHAEKAYARESDGTWGDNGYPWSASAVGGGT
IFYAGISFRYKDVDFDAREYVAPDALDPQWPIGYSDLDPYYNKIEQWLGVSGDLNADPFKNSAGTGSLRTAHSYSQQGAL
IDQAGKELGLHPFPTPLAVNTTDHNGYPGCDNLTSCTDYACPIGAKADSFTRILKPVLKLDHVTLMKNTKALAFHQTDPS
KIDSLECLDLATKERFHIRAKQFVLAGNAIQSAALVLRSTNKWWPDGVGNKSGLVGSGLSFKNSEYVSGWVDNHPYEQLN
EPLKGLYSTVSITDYYQDPHCHSGLGGLIYEANPWTYYGHEGSGMYVQLECIIADQPMVTNKVRLSNDKEATGVPKIVLD
YQTHEWDKRRLSFMIERAGEILTAMGAKQIQCNPSYYYLGSAHLHGTCRAGTDELTSVVDKFGRFHMIDNLFAADGSFMP
FAGGVNPTLTIQANALRIAENMSV
>lcl|NZ_CP026520.1_prot_WP_042226719.1_3639 [locus_tag=PC41400_RS18440] [protein=hypothetical protein] [protein_id=WP_042226719.1] [location=complement(4064150..4064398)] [gbkey=CDS]
MDNKERNTLYQVIYAISGVTGEDGDELVETFKQGPLEVDKRDFFEIVQRVESLFDCTLDMNLEGPYLIHADEIVTKITKL
NV
>lcl|NZ_CP026520.1_prot_WP_042226721.1_3640 [locus_tag=PC41400_RS18445] [protein=histidine phosphatase family protein] [protein_id=WP_042226721.1] [location=complement(4064415..4065056)] [gbkey=CDS]
MRLIIIRHAQAHCNILEDDALMDAYDPHCELTETGIGQAVKLRDEYPVSLTPSVIYSSPLKRARETAAIFHQRFPAAPFI
EDDRLSELKAPESFVPPITQGQWDLYLEQRIQSPHLEIVKGLESLDVQRERIEEFYKDLFGTYAGEACNIVIFTHAFSIQ
LSILYFLGLGNEQLLQWQIKASNTAMHIIHYDSMSGSFLLESLNNRSHLQTTG
>lcl|NZ_CP026520.1_prot_WP_042226723.1_3641 [locus_tag=PC41400_RS18450] [protein=LLM class flavin-dependent oxidoreductase] [protein_id=WP_042226723.1] [location=complement(4065095..4066120)] [gbkey=CDS]
MIALDPIQYYCQIPGVGHYASKKEFMENSIKIAQLSDAYGIVGSLSFFNHSVLDPWAVSSVIMQHTERHVPLIALQPYMY
PPYTAAKLIQSFTYLYNRRIDLNMITGAVTGELQQTGGYIDHSSRYKKLHEYVQVLRLLLESDSPVSFKGDYYELNNLEF
KPPLPDKQLFPRIFMSGSSEEGLETGLKAADFVVTHPGPLEHFKRHFSDKVQGSAVQSAIRIEIIARESAEEAWKVAHAR
YPGNRQGTIQLRMKKNSESSWQRTLAELAIASETYDGVFWMGGYMNGGIYSPVLVGDYEQVAAYLNEYYKLGVKAVLLGS
MYSEEDFIHFSRVKEGISNPV
>lcl|NZ_CP026520.1_prot_WP_042226725.1_3642 [locus_tag=PC41400_RS18455] [protein=type III PLP-dependent enzyme] [protein_id=WP_042226725.1] [location=complement(4066148..4067434)] [gbkey=CDS]
MNLDHAEITALTKEFETPFYLYDGDFIEAHYRQLRSRTNPAIQYYLSLKANNNIHVAKLFRQWGVGVEVASAGELALARH
AGFSADNIIFSGPGKKRSELQAAVESGIYCIIAESVQELLYIEELAEQANKTVRVAVRINPDKSFGSTAIKMGGVPRQFG
IDESMLDTVFDAIGSLRFTKFIGIHVYTGTQNLNTDSIIESMKYTVDLGRSIYERYGIVCEFINLGGGFGVPYFAHEKAL
DIGKITDEVSDYVQEARDTLFPQTTFIIESGRYLLAQAAVYVTEVLYRKTSKGENFVIVDGGMHHHAASTFRGRSMRSNY
PMECIPVSEDPGRRELEKVTIAGPLCTPEDCVGKDVQLPALYPGDLVCVLNSGAYGLSFSPVHFLGHPTPIEILKRNGLY
ELIRRKGIPEDLVVTQLQTEPDLLFADK
>lcl|NZ_CP026520.1_prot_WP_042226728.1_3643 [locus_tag=PC41400_RS18460] [protein=ATP-grasp domain-containing protein] [protein_id=WP_042226728.1] [location=complement(4067463..4068722)] [gbkey=CDS]
MKFTHPLDYYRLSGKQILWYMNIGDEQDGQASNYFPSVKDLQSEKIVVQQEHQLLFLARPQDTVFFHAMPEPAFLDYWKE
RNLVLPTIICCDELAQVPDLARHTIIPFIVSERLLELKERYPQMDIIAPELTVCREINDKFSTRRRMERNGFNVTTGYFC
SDIGSLEHAYDQLVSAGFSKCVLKVPYGSSGKGLKVIDNERNFRFLLNYIQNRQTSVDLLLEGWHPHRLSLTSQLFISEY
EVHLLAVTEQIVDPNGVYKGTNFSPSLSQSESAEYREEILRAGELIRQMGYRGVLGIDSIIDTNGELIPVIEINARLTQV
TYILPLVIEQKKRYEFVESRVLVFNSGMDLGFEDYVNDLSEVTRDLPVHIDLYNFCKASGAFKNTYKLFVLVSAHNTEQL
TKARSLLDELNTRMTTGVH
>lcl|NZ_CP026520.1_prot_WP_042226731.1_3644 [locus_tag=PC41400_RS18465] [protein=acyl carrier protein] [protein_id=WP_042226731.1] [location=complement(4068719..4068982)] [gbkey=CDS]
MSLETKENVRTAQVISCVNEILQTKDPITPETDLFLLGLDSVTAVNLIVSLEGVFGITFNDDELIVENFSTVRKITEHVT
EKLEGTL
>lcl|NZ_CP026520.1_prot_WP_042226734.1_3645 [locus_tag=PC41400_RS18470] [protein=BtrH N-terminal domain-containing protein] [protein_id=WP_042226734.1] [location=complement(4069052..4069960)] [gbkey=CDS]
MCLTRYDEKFFDCRKSQIIAYLDSNRVPVIPLFYNSYQSTAEIYRQIFIENKSKWKYSEPSFSDDDLLRKGITPVRASFP
DFSQASDCLKELLARRKLVFVWGDEYYLPYRKEAFHVIHSTHSFVVTDYDGENKAYYVEDWDGLYGYLPAAHVEAAFDSL
SGQMRTLLVLELNDEEMRENKQEDLALFRQWLQAFEDDYIFYDRVLLEMRDYEENRLISMDHGLRLIAASRHVFSKFLHY
IDDAPEEADLLIRNHQLANHIATIVRRYMIARQIDWDGAASKIRQLREQEADFMRKLKRRYA
>lcl|NZ_CP026520.1_prot_WP_042226737.1_3646 [locus_tag=PC41400_RS18475] [protein=gamma-glutamylcyclotransferase] [protein_id=WP_042226737.1] [location=complement(4069941..4070360)] [gbkey=CDS]
MPNLFVYGTLREGENNHKYMKDATLLSRKASIAGSLVDTGNGYPGLLLENQRVAGEWYEVSEDTLKRIDELEEYFGPGDT
RNLFDRIECQVTENGGTHAGWTYVYNRDDYLETRFSDWKQYRLQHASGVEEKQDVPHSL
>lcl|NZ_CP026520.1_prot_WP_042226740.1_3647 [locus_tag=PC41400_RS18480] [protein=SDR family oxidoreductase] [protein_id=WP_042226740.1] [location=complement(4070434..4071132)] [gbkey=CDS]
MIVFITGANRGLGLSLCEVFLERGHHVLAGVRNRNRAERLQPLITKYKEKLSIIPVDVSSEDSILQAYSLIASVTPKLDA
IINNAAREGIRCKKIREVDIADFEETWKTNVLGPMAIIKHLKPMMESSPVKSILNISSEAGSISELPLKTYGYSISKAAL
NMFTRMLERELKPENFKVLAVHPGWMKTEMGGEDAPYDPGEIARHLYERLTESSGKEHDFNIVDHLGNRMAL
>lcl|NZ_CP026520.1_prot_WP_042226743.1_3648 [locus_tag=PC41400_RS18485] [protein=zinc-binding alcohol dehydrogenase] [protein_id=WP_042226743.1] [location=complement(4071144..4072193)] [gbkey=CDS]
MEILTYTGPSQLIVETAEELPLKSGECRIQSLYSGISHGTEMGAYRGIAPYFTRDMDSETRLFEDLAEEHKVKYPIRSCE
DDAWFIGYSNVGKIIEVGADVEGFSVGDIVFSHGRHQTIVCKHHKKIFKLPQNLEPELGIFYTNLMTAYNAVLDTRIKIG
DVVVVSGLGVVGQLIAQLAKLSGATVVGVDPLENRLNIAKEVEIDHVFNPNRTDVAKEIREITNNRGADAVFEVSGHSTA
LNQAIRIAAPDTVITAVGWYQGEQSVLNLSEEFHQNRITIRASQTLGIDPSISHMYNDARRRNIGKDLLLKLKLGNLISH
RIPFGDAPKAYDMIDKHPHEVLQVVLTYS
>lcl|NZ_CP026520.1_prot_WP_042226745.1_3649 [locus_tag=PC41400_RS18490] [protein=PIG-L family deacetylase] [protein_id=WP_042226745.1] [location=complement(4072217..4073044)] [gbkey=CDS]
MNQDKRAFMFISPHFDDVVLSCASTLMELVNQGHTCKVLTVFGGCPSVRFQPGEIARQYAAEDLGLFEDEIEGEHLSILV
ARRLQEDQRAFRHLAGVQVEVLTFPDAIYRENKGQPYYRTEEDLFGIPDKQDEADFLPKIETYLQSCDLARKYTWVFPAV
SKHVDHQLLTKAGLRLMSQGYPVLFYSEFPYWQQYNEFSQDGWHQLELRNSVYMPVKRAAVLEYKTQLLGLFGEEAETKI
GSGGVLSETELFWIQETDTQVWPVFRAISPEPLQT
>lcl|NZ_CP026520.1_prot_WP_042226747.1_3650 [locus_tag=PC41400_RS18495] [protein=2-deoxy-scyllo-inosose synthase] [protein_id=WP_042226747.1] [location=complement(4073047..4074153)] [gbkey=CDS]
MTTKQICFADRCFNFAFGEHILESIEDYIPRDEFDRYIVISDSGVPDSIVHHAAEYFGKLAPVHILRFQGGEEYKTLSTV
TDLQEQAIALGANRRTAIVAVGGGLTGNVAGVAAGMMFRGIALIHVPTTFLAASDSVLSIKQAVNLTSGKNLVGFYYPPR
FVFADTRILSESPPRQVKAGMCELVKNMLILENDNLEFTEDDLNSTNVYSPKQLETFINFCISAKMSVLSEDIYEKKKGL
IFEYGHTIGHAIELAEQGGITHGEAIAVGMIYAAKIANRMNLLSEHDVSAHYWLLNKIGALQDIPLKSDPDSIFHYLNHD
NKRGYLKLDEDNLGMILLNGVGKPAVYNQSLLTPVRKTLIKEVIREGL
>lcl|NZ_CP026520.1_prot_WP_042226749.1_3651 [locus_tag=PC41400_RS18500] [protein=aminotransferase class III-fold pyridoxal phosphate-dependent enzyme] [protein_id=WP_042226749.1] [location=complement(4074156..4075454)] [gbkey=CDS]
MKQDMVENSEKLLAVLGTHIDSPVNPFRNERVMFSRGSGAYLFDYDGGNYIDLMNGKGSIILGHNDPHLNAALRSFLEQD
REIVTGPSKPIVELAERIRKDSALPDAKVSFYATGTAACRAAVYAARDYSGKKIVLSSGYHGWDPMWRQQGALLEPNEDG
VIEFYFIPELLERALTEHKDQVALVLFSPDYTYLSASTIERILGICRAHGVLVCCDDVKQGYRHRQGSSLEMVTAEKADM
YVFSKGLSNGHRISCLVSTDEIMAETKEHTYTSYYQMLPILSSLETLKKMEAGKGYDLIRFYGHALTGNLKELFAQSALP
IEVNGSSIFQLVFGDEKLEEAFYREAFNQGLILFEGDNQSLSLCMDKDVQVDMIRRFTNVTDVLSEQYKHLRGKEVTAEQ
TFRTAWNMIDGASDLLPYEKQLKLLDNLIGGG
>lcl|NZ_CP026520.1_prot_WP_042226751.1_3652 [locus_tag=PC41400_RS18505] [protein=hypothetical protein] [protein_id=WP_042226751.1] [location=complement(4075496..4079170)] [gbkey=CDS]
MNSFGIKCERLLELRNNETFTAHIPPLYPIDPKPAVEFLEQYSGTEEELPGQMARYIEQREPSRFDQIREVCGDLPWIVR
SSGEEDLNETTNAGAYLSLVCEHRDNFYDVLAQVMLSGSTEHARNQNRRFVNRNLHDHEPIPVFVQPLVQSEHAGLHVPH
DATPYVRGEDLQTIIGLLKNLHSTMPDYALDCEWVGDSDFGTISMTSLTELAGGNLTGQIAFGFDLFSTQHVKANNSVRY
LLPGDELNLWQGHSFQEVNMQSLFLVQTRPAKNYRLFQPVYELSDSSQKKLEKDCALCLDAKEFISQGGKPFFGKFIKAV
TLNEAWNAYLQYTSKDVKEDIAAILVEFGTRTEHAGIMFNQVGIPVIRIEIALIPESMNYIVLDPFSLQCRLYSHNKEAK
ALEYGTREIIPLPDNCILVQDREEKKQPEAMVVTADEYFNELLNKPFFSEKTRQHLAANSAYPDYSFIFHGKDGVRSPSY
YANAADRLTGETAPAFLESREWPEASAKYFLALFHAKNLLKDRNIAAYFERQAEDSGGSKETLVYWKVLRIHPLLAKCSS
GDRETLLERVRAEMNANSAVIELRLDVLFLFEQYLDALCIYSKPEEEELIQSFIGILSSVKDEDLPIVLDLARNGIFNAG
HLCQFLKHATENRQLFDLYRNYSLVSKEFAVIPYDLSVIGRFPYFNEVFVELNNEMNKYASLGELSSSIFHTVVETYDKS
AKEILLSVIDDGSPSYYEAYTRLLTEWLGLIRQFDKNEKVIQTFLEWIEIQQDRQAEIEDYFLEEIHWKTRIEEAEAGNA
ENLSLTNLHQLHNVLHQWSLHLVPASSGAHMPLFVKEMVAFAQSFSEQENRLLRMQKDFFEIELAMCTHKAGFMFYKHSV
SVEFSEPPSVHDDEIARLSAFASFMHKCNDWFEAYRFRTHYEKVAGTWTCYMTVKSSEDGALKQSDYKTIFHIIRFVLDS
SYDFSHNPLSAVEDLSAQFDEPEWKDIFGKLVNYRVRFDDEGQFINLKLFAMSTFFTYLCLSPSLRNRLLSLYREGFDRM
VGELAVLEDGMARADHYSEWLEHYEAATYIALFLAAIYPLETVGKLESLYERQLTYDIVSRNLLKHKDAAGKVIETLPSW
DKALQSSVTDKLLHYCPAKYLKSVSSVNDVAKELSKRGKSFKRAKQVLLNAYADSLEPDMLQPFIDDLQYVPYAVTEAQN
QRLDKCLTDPGKKRYDIQKEILYI
>lcl|NZ_CP026520.1_prot_WP_042226753.1_3653 [locus_tag=PC41400_RS18510] [protein=hypothetical protein] [protein_id=WP_042226753.1] [location=complement(4079217..4081031)] [gbkey=CDS]
MKDNPISLFHELESYVRGSKDRLHHYMYACILEQRLADLLLLSRCGVHPEDSDRLAVRLASQRSLCVQAASDIAEGTQEA
WGSTPAWPDYKVAKSSVTQKRPELFSLIRVMQELRTPECSGYDLCDLEVCREDALHWLRDHADPSSPPVLIGVRTGGAFY
APLWASALRQRWGSDAHYHTVRALRKPSDPGASLYLPEELSMLPDAIDPDTDIVILEDQPHTGGTVLELAGRLSVKYRLN
KSVWVSSPGRLFKIENGRLVKRSDRIPLNAAKKKRIWQMLGSSEEVVEFLLPKLGISDARPADLETVPSKSVPHWNDPMY
RHDAPFRINPKKTPFFIRQKSTNEPVAFAKFIGKDLFGDFQFHQLKKFEQYLPDVLAYEDGYVVTKYEPELREMRDLFVN
MPSAVTNEICQNMSGYWKTLLASCQVSGGHPVHPLADHWQAYLGEMEDFIGRKLPSDPDWFEQSLHTEWTSSAHVYTSLP
YANQYGHWRARLSAGQRLKTYRFHVDSTWGGTSSIEVELASFLLENRVRPDDFRLLVNQVQKEAESLTVEAVADALPIAC
ILQVANLLKQAKGESRLSKELILSETMQSFEYLQAMKPKLSDVR
>lcl|NZ_CP026520.1_prot_WP_042226755.1_3654 [locus_tag=PC41400_RS18515] [protein=glycosyltransferase family 4 protein] [protein_id=WP_042226755.1] [location=complement(4081021..4082259)] [gbkey=CDS]
MNILRLTPHYYFETTEWPSRFDPMGGMQVQITNLTEWLADQGVRQDVLTTGIPGIPRTLPIRERLTVHSVRFMTLPFKSS
QTGTLFLDQSWFMGAVKWIVLNGKKKNYDAIHVHASGVVWPLLAGMFAQKFLKKPLILTIHCSRIFTYKPMNKWDQFVHD
FVKSVELKSIKLSHKAVVLTDKRLDSYNRLLEDSSKVTAISDCIGSNHLSHSIDCPFCNRLKTELLGKKTVLFLGRIAHE
KGWATFVSVAKELADKIGDLQFIVCGDGPQREAMEEQIKAANLQNQFRITGFISHKFVSCYLHHAQLFLLPSHHEEFGGS
LIEAAIAGVPIISTNNGGPADIFTHGETAILTDPSDVKGIADEAYKILTNDSVAESLRLHSRPEVVSKFLPHCVYPNYLN
LYASKEAAVHEG
>lcl|NZ_CP026520.1_prot_WP_042226757.1_3655 [locus_tag=PC41400_RS18520] [protein=SPASM domain-containing protein] [protein_id=WP_042226757.1] [location=complement(4082278..4083030)] [gbkey=CDS]
MDKLFSMIEVEVNSQCNRTCWYCPNSVSQRKENGEMDPALYKSLMEQLSSLDFAGRISFHFYGEPLLCKNLDLFVGMTTE
YIPRARPIIYTNGDFLTEKRLKTLTELGIQKFIVTQHAGAKHKFRGVYDQLAGAEKEKVVYLDHSDLILSNRGGIMDNIP
QASKANMSCMVPSNLVVVTVLGNVLPCFEDFNQKMVMGNIGEHHISDIWHNEKFTSFRKLLKEGHRGKSDLCKNCNNVSV
QTEDQYDYVL
>lcl|NZ_CP026520.1_prot_WP_042226759.1_3656 [locus_tag=PC41400_RS18525] [protein=DegT/DnrJ/EryC1/StrS family aminotransferase] [protein_id=WP_042226759.1] [location=complement(4083060..4084316)] [gbkey=CDS]
MTIPFDQWPEWPQHSDRTRQKIEEVFQSNRWAISGYWTGEESMERKFAKAFADFNGVPYCVPTTSGSTALMLALEALGIG
EGDEVIVPSLTWIATATAVLNVNALPVFVDVEEDTYCIDPQLIKSAITDKTKAIIPVHLFGSMANMDEISEIAQEHNLFV
IEDCAQSHGSVWNGQRAGTIGDIGAFSCQQGKVLTAGEGGIIVTKNPRFFELIQQLRADSRVYCEDSSELMHGDMQLVKK
GDIQGSNYCLSEFQSAILLDQLQELDDKNAIREKNAMFLNDALSKIDGIKVMKRPPQVSRQTYYGYVFRFDPVKFGGLNA
DQFCEILREKLNMGTFYLHPPYLPVHKNPLFCPWTKNRYLKSVRKTEAYWRGLHYPVSERASEQSIVIHHAILLAEPSHL
SLLIDAVAELARKFCVTH
>lcl|NZ_CP026520.1_prot_WP_156972782.1_3657 [locus_tag=PC41400_RS29880] [protein=hypothetical protein] [protein_id=WP_156972782.1] [location=complement(4084968..4085138)] [gbkey=CDS]
MIGGRTIHLDIREGWEAAVERPFCRSRGSCREPAVFSAAAEQEEIGEILNSLQIQD
>lcl|NZ_CP026520.1_prot_WP_042226762.1_3658 [locus_tag=PC41400_RS18530] [protein=suppressor of fused domain protein] [protein_id=WP_042226762.1] [location=complement(4085250..4086311)] [gbkey=CDS]
MSTEPNTAGWDAIDKEFVKLYGDQEPKHYGTLISYALGGPDPLDGISAYKTDEPAAHWHFVTYGFSELYDKESDNPEESG
YGFELTFRLDRGAEEEEPPAWAINLLQNMGRYVFNSGNIFRSGDYMDANGPICLGSDTCLTALAFIRDPELSAIDTPNGR
LEFIQMVGITRDELEAMQTWNTLGMLSACLPYMPKYITNLSRDSLLKVETVSEAAEKGMREDGSNTGFLFVDQLEWEPGR
KGWLSKAPAVLTVGAKQAGIIGKLLQGRILKGKDLTLVGPEVRVIFKAGEEPGYAAGEGEVRIILNEQAAGELSRGLVPK
KSVLEIPSLQRIVIRIVKTDIKDSEGNVVEVIG
>lcl|NZ_CP026520.1_prot_WP_042226763.1_3659 [locus_tag=PC41400_RS18535] [protein=NAD(P)H-dependent oxidoreductase] [protein_id=WP_042226763.1] [location=complement(4086436..4087053)] [gbkey=CDS]
MGKESDVCLKTLVVVAHPNLQQSRVNKRWVEELRKHSGEITVHELYGEYPEKVLDIGREQALLAECERLVLQFPLQWYST
PPLLKQWLDEVFTMAWLFGPGGKAVAGKELLLAVSIGGAEETYEAGGLIGYTISELTRPLQAFASQIGMTMLPHFKFHGA
NQAADEQIESSAGRYVRHILTPELDPRIARVRMLNEMKQKMQASL
>lcl|NZ_CP026520.1_prot_WP_042226765.1_3660 [locus_tag=PC41400_RS18540] [protein=TetR/AcrR family transcriptional regulator] [protein_id=WP_042226765.1] [location=complement(4087101..4087754)] [gbkey=CDS]
MSPRTKEQNEAIRNQRKQEVLQAAIQVYVDKGYAAAEMGDIADRAGLAHGLVYYYFKNKKALFRELYETMMEESKRFTKS
YFEQEGAVADLFEGYARVVCDRVRVNPVIPRFYMRVSLDVHHLYTPEEFSPFEWVKSFLQEMTQAIEKGIGEGTIRPGNS
DLMARQFWGAVSQGMNYLDQAQQEWMANGMPETEKKEQWEKALNQVVESSVAMLRVR
>lcl|NZ_CP026520.1_prot_WP_241688694.1_3661 [locus_tag=PC41400_RS18545] [protein=iron-hydroxamate ABC transporter substrate-binding protein] [protein_id=WP_241688694.1] [location=complement(4087911..4088705)] [gbkey=CDS]
MPKKPKRIVDTTSFYTGYFLAIGVKPVGISEGAMDNPYFKEMLDGVRSIGKEPSPESILELNPDLIVTFTGTKDIDKLEQ
IAPVVAIDYGKKNYKDQLLEFGKLTGKEDKAKEWVDRWEQKISGLKPQIQAAVGNKTVSILNPYAKGLYVFGHNYGRGGE
IIYGEFGLKAPPEAQKAAIDSGTGWASISLEKLPEFTGDIVFTCPWSGDTSDPGKVYENPVWKGLPAVKKGHVFQLNPKG
DTFNDPVSLEAGLKFISESLLSVK
>lcl|NZ_CP026520.1_prot_WP_042226768.1_3662 [locus_tag=PC41400_RS18550] [protein=DUF6022 family protein] [protein_id=WP_042226768.1] [location=4089193..4089738] [gbkey=CDS]
MPTTKALSDFLKEQPQNKIGAVAAFIESHIRENWQEILVKENDRLLKAYAEAGDPAYGTYLNLLLLPVHRQLNEAGLRPE
PRLPGNFDISREWGRNPDQSDQERWMWSTLHPTEGPPVGTIVTVTYHDHTRFRIPRQPGIIALTETGQQEIVEVLSRMSA
DFKQALEFSVEYENYLRSLEQ
>lcl|NZ_CP026520.1_prot_WP_042226771.1_3663 [locus_tag=PC41400_RS18555] [protein=NADAR family protein] [protein_id=WP_042226771.1] [location=complement(4089906..4090472)] [gbkey=CDS]
MNYTIEELRKAYQAGDRRKFLFFWGHTPPADGSINSSCFSQWWRSGFIVDDESYTCTEQYMMAEKARFFGDKKMLEAIMN
AGHPKEMKAFGRAVASFDANVWESRCYEIVKKGNLAKFRQNDDLWSYMRATKNRILVEASPVDRIWGIGMAQDHPDADNP
MKWRGKNLLGFALTEVRDILLQEEAERN
>lcl|NZ_CP026520.1_prot_WP_042226774.1_3664 [locus_tag=PC41400_RS18560] [protein=NUDIX hydrolase] [protein_id=WP_042226774.1] [location=complement(4090469..4091359)] [gbkey=CDS]
MKRDRNGLTEEQFLAQYRAGDYVRPSVAADMVILTVADTTELNYRKLPKKNMRVLLIQRGEHPFLGKWALPGGFVRPGET
TEQTARRELREETSVDNVYLEQLYTFSEPDRDPRSWVMSCAYMALIDSSQVKIAAGDDAAEAAWFDVTCRLLSEEKLRLE
DGYSITRRYELKLTGERDTLTSVVEQRKQSAKASSAVEYAVAASGELAFDHAKIIACAIERLRGKLDETDIALHLLPELF
TLTELQQVYEVILDKELLKAAFRRKAAPLVEATDQYTENAGHRPSRLYRRNREELE
>lcl|NZ_CP026520.1_prot_WP_042226776.1_3665 [locus_tag=PC41400_RS18565] [protein=YdeI/OmpD-associated family protein] [protein_id=WP_042226776.1] [location=complement(4091603..4092187)] [gbkey=CDS]
MSKKAPELPVLCFENQEAFEIWLESNHETSRGILLQIAKKNARLVTVSYQEALESALCYGWIDSQKKTFDDQSWLQRFTP
RGNRSIWSKVNKDKAEQLIADARMKPGGLRAIEDAKRSGQWDKAYEPQSSGSVPEDFAARLELNARAKAFYDTLNKQNKF
AIHFRIQNVKKQETREKRILQFVEMLERGEKIYP
>lcl|NZ_CP026520.1_prot_WP_042226779.1_3666 [locus_tag=PC41400_RS18570] [protein=UbiX family flavin prenyltransferase] [protein_id=WP_042226779.1] [location=complement(4092269..4092835)] [gbkey=CDS]
MKVIVGISGATGAILGIRLLEGLKEAGIETHVVISPWAAANIQLETAYTVKEVEQLADYSYAHKDLGAVISSGSYRMDAM
IVAPCSMKTLASIRLGLSDNLIARAADVLLKERKKLVLLTRETPLSDIHLEHMLALSRMGTTIFPPVPAFYNHPQNIEDL
IRHIVVRTIDQLGVSLPAAKRWGGMKEA
>lcl|NZ_CP026520.1_prot_WP_042226782.1_3667 [locus_tag=PC41400_RS18575] [protein=UbiD family decarboxylase] [protein_id=WP_042226782.1] [location=complement(4092904..4094397)] [gbkey=CDS]
MKKPIKSFRDYVETLDRYGDLFTVDQETDWNLEMGAMIRYAYELPSPAPLFTKVKDSAPGFRVLGAPVGISPDKRHPYLR
IALSLGLPAHSTLSELVEEWSYLPDAEPIPPRVTADAPCKANKLFGEEIDLTRLPVPYIHVGDGGRYLNTYGVMIVRSPD
GSWVNWSITRVMLHDERTMAGVIVPTQHLGMIYEQWKEIGQPMPYALCLGADPGIAMIAGYPLPDGTSEAELLGGWYGEP
VEVVRCETNDLKVPATSEIVIEGVVSLQETVPEGPMGEYAGYTWVGHRKPVPCFKVTAMTHRDDPIMPVVAAGVPPEENH
TNWGVAIAASIRHELRKRRLPVTACLIPFESAVHWLVVTVNSHAEEAAEMTGDSLARQIGEAVFSSRGGSYIPKVIVVND
DIDPGSIEQVVWALATRSHPDKALLFPGQKVLPLVAYLDAHEKEAGTTTKVVYNCLSSDEWPKDYRPLEASFAGYPDELR
EQVRERFKSCGFTGERG
>lcl|NZ_CP026520.1_prot_WP_042226785.1_3668 [locus_tag=PC41400_RS18580] [protein=LysR family transcriptional regulator] [protein_id=WP_042226785.1] [location=4094535..4095434] [gbkey=CDS]
MDIKKMYYFMTIVEEGQITKAAQRLHMAQPPLSLQLKALEEELGVTLIDRENKRFEITPAGWTFYKRAKEILHSIDGAIA
EIKEQQNGLRGTLSIGTVMSCVPYLPSVMKQFRLGSPRVSFQLWEADSFRVEELLQNKVIEMGIVRLPLQSYLADRMPRV
SIHRLQTEPLVAVFPPEWKVPDGTSVPVSELASSPLLILRGQGDYGVYHKFIETCKTSGFEPNLVCESPDVSTLMILTDS
GLGISVVPRSALLLKKPGQVRYAEIDPPIESDTAIITLEHQFVSQAARNFKELLIGSLQ
>lcl|NZ_CP026520.1_prot_WP_042226788.1_3669 [locus_tag=PC41400_RS18585] [protein=VanZ family protein] [protein_id=WP_042226788.1] [location=complement(4095569..4096174)] [gbkey=CDS]
MYQGTLRKMTLLLLGLYTGLIFYFLYIGFNRSSHIVDRGMRYNLIPQGIPLHYPMGRDFQSWFFEFGNFAAFVPFGLLIP
LLFRCKFVPFIVCFLASITCLETVQMVTRLGAFDIDDIVINTLGAAVGFWAQRLAPGRRDSLQGIVRLVWITIFLSAGTV
AAVGGMNVSMEKGGDGYLFKPQKSPSHSVVLYNSVQPKRAE
>lcl|NZ_CP026520.1_prot_WP_042226789.1_3670 [locus_tag=PC41400_RS18590] [protein=metallophosphoesterase family protein] [protein_id=WP_042226789.1] [location=complement(4096453..4097133)] [gbkey=CDS]
MKRTLVVSDIHGEIDKFTRLLEKVSYNPRQDQLILLGDYVDKGPDSRRVVEKVMQLHREGAIVLKGNHDHMMVKSFERDP
VFIERWFRNGARATLTSYEHPGAASETPAPDTLELTPLLEEHLAFLGGLQTYYETEDTIFVHAGVHPSTSLAETDAYVLM
WIRDEFHKGYRGSKTVVFGHTITETLHGKKEVYFGDNNIIGIDGGAVYGGLLHCLELPGRNVYSVD
>lcl|NZ_CP026520.1_prot_WP_042226791.1_3671 [locus_tag=PC41400_RS18595] [protein=beta-eliminating lyase-related protein] [protein_id=WP_042226791.1] [location=complement(4097336..4098436)] [gbkey=CDS]
MNDKESLKAAFKRAPYQLGGNGPRDIGILKQVFDEIEGTQESDIYGAGTLIEAFQSKIAAYLGKESAVFFPSGTMAQQIA
LRIWCDEKGSKEVAYHPLCHLEIHEEDGLKELHHIKPVLLADKDRLITLEDVKTLDGEISCLLLELPQREIGGQLPAFEE
LEAISAYCRERGIKLHLDGARLFEITPYYQKSAAEICALFDSVYVSFYKGIGGIAGAILAGGSDFTERSKVWKRRHGGDL
ISLYPYIMTSDYYFDRRIGKMAQYFGEAKELAAIFNECEGVYTNPEVPVSNMFHIHFKLPETAVLPILREVCEATGVGVT
SWLHEREEGTCYSEISVGDSYESVPKEKIREAFRLLKEKLEAALSA
>lcl|NZ_CP026520.1_prot_WP_042226794.1_3672 [locus_tag=PC41400_RS18600] [protein=NUDIX domain-containing protein] [protein_id=WP_042226794.1] [location=4098614..4099114] [gbkey=CDS]
MGMSDYYRNLREKVGSEWIFVPSVAGIVRNESGEILLQNKGNGEKWSLPAGAIELGETPAEAVVREVREETGLYVVPSKL
IGVFGGKEYRYQYPNGHKVEYTILMFDCVIHSGTLNPLDSETAELRYFSPEQMPELALPYPKQLFSREEKNEVYFQWKDD
YLKNLM
>lcl|NZ_CP026520.1_prot_WP_042226798.1_3673 [locus_tag=PC41400_RS18605] [protein=MFS transporter] [protein_id=WP_042226798.1] [location=4099550..4100986] [gbkey=CDS]
MNIQTDSKNKRQLPLLLVLVFSLFTACMTADMVNPVLPLIGKDFAASDAQAGWVVTGVMLVIAIGVPIYGRISDYFSLRR
LYPLALLIFAAGSLVCAAAPTLLVLVLGRMLQGAGSAAIPVLSVITVSRVLPSGNRGAALGILAGSIGIGTAVGPIFGGL
VGQLFGWPALFWITLVLLLALIPCAVCVLPGGRPGTETAKDRRFDLLGGIVFGLAAGLFLIGVTQGQAEGFRSLSSLGSL
AGASLMTVAFVWRIFTAEHPFLPPALLANKGYLAAVLAGFFTMFAYVASLVYIPLLIVEVSGLSPGAAGLVLLPGAAAVA
VCSPAAGRLSDRIGAKLPVTAGLLVMGISLLFMSTFAAGSSPLLASIGMLGLGFGVAFTNSPANNAAASTLENEQLGAGL
GIFQGALFLGAGTGVSVIGAFLAARKEAGQDPLNPFYVLGAAPYSDAFLAIAVAVIIALVFSAGLQSRKQKKSSETSR
>lcl|NZ_CP026520.1_prot_WP_042226801.1_3674 [locus_tag=PC41400_RS18610] [protein=ABC-F family ATP-binding cassette domain-containing protein] [protein_id=WP_042226801.1] [location=complement(4101134..4102672)] [gbkey=CDS]
MSLLEITGMSHSFGDRTLYKQVDFELFKGEHLGVVGQNGTGKSTLLGILTGEIIPDEGTVRWQSNIRIGYLDQHAVIDRN
RTIGEYLKTAFASLFELETKMNALYEESASTGREEPLLKAAHYQEELDRQQFYEVDSTIQKVVNGLGLNAIGTGRPIQEL
SGGQRAKVILAKLLLEKPDVLLLDEPTNFLDKEHVSWLADYLTSLTNAFVVISHDAAFLDKITTGICDIEGKSIKKYPGR
YSHFLRQKEHLREDHIRQYNAQQKTIQRTEAYIRKNMAGVNSRIARGRKKQLDKIERIEAPAFVAKPFIRFRELPLGSQT
ALSVNELEAGYAEPLLPKLTFTVRNGEKIVITGFNGIGKSTLLKTLMRKIPALSGGFRFGEHVKTGYYEQDLIWEDDALT
PIQFIADGYPQLSNKEIRRHLAQCGVNEARATQTIRTLSGGEQSKVKLCGLLLTPCNVLIMDEPTNHLDAEAKDALRDAL
IHFGGTLILVSHEQKFYEGWVDRVLEIGSRKG
>lcl|NZ_CP026520.1_prot_WP_042226804.1_3675 [locus_tag=PC41400_RS18615] [protein=proline--tRNA ligase] [protein_id=WP_042226804.1] [location=complement(4103074..4104795)] [gbkey=CDS]
MRQSRLLLTTLRESPADAEVISHQLMLRAGFIRQLAAGIYTYLPLGRKVLRKAEQIVREEMDRAGAQEMLMPVMQPAELW
KESGRYSVYGPELMRLNDRHEREFALGPTHEEVITSIVRGEINSYRKLPVTLYQIQTKFRDERRPRFGLLRGREFLMKDA
YSFDADWEGLDESYRQMFQAYNRILTRCEINFRAVEADAGAIGGEGGTHEFMALADIGEDTIAACTVCDYAANLEQAESR
AARTDSVERSEEKAVKFHTPGIRTIGQLVQSLGIRSSDIMKTIVYTADSRPVAVVVRGDREVNEIKVKNYLRAGDLSIAD
AEAVEEATGAPAGFVGPLGLSIPLLVDEEVFGMANGIAGANEADYHYRNLDPGRDLAESEVGDFRQAAEGDRCPRCAEGV
LKLSRGIEIGHVFKLGTKYSESLEASFLDAAGKSTRMIMGCYGIGVSRILSAVIEQNHDTNGIIWPLALAPYQVHLIPVS
IKDDVQVQAANDIYRRLQENGIEVLMDDRDERPGVKFKDSDLIGIPFRIVIGKDAGQGKVEFVERRSGLKETVTLQAACE
KILEQVRLSKPAH
>lcl|NZ_CP026520.1_prot_WP_042226806.1_3676 [locus_tag=PC41400_RS18620] [protein=GNAT family N-acetyltransferase] [protein_id=WP_042226806.1] [location=complement(4105347..4105901)] [gbkey=CDS]
MDIRIREWTADDEFEGTEVDGSFIVDSVLVLTLEDGRIGYAVEEVPRYTKRYEDEPDGETEDTGYAGYIGNPDQVMYLAC
IGDKFAGQIQLRKNWNGYAYVEDIKVDAAYRRYGVGRRLIDQAKNWAKAGGMPGIMLETQNNNVRACKFYESCGFVLGGF
DACLYKGLDKQNNEIALYWYLMLD
>lcl|NZ_CP026520.1_prot_WP_042226808.1_3677 [locus_tag=PC41400_RS18625] [protein=GNAT family N-acetyltransferase] [protein_id=WP_042226808.1] [location=complement(4106079..4106867)] [gbkey=CDS]
MNEPGIVTLNTENIGGEHICCAISDKKCANGVKLKKEWLQQRFQEGLRFKKLDVRQKVFIEYIPAEYAWCPVEAPGYIFI
NCFWVAGSYKGMGYGKKLLGECLEESGATNGAIAVTSRTKRSYLSDKSFFTYNGFEVCDQAPPYFELVVLKFKEDAPTPK
FKQSVHEQKTDVVKGFQVLYTAQCPYTDHYVGELEDCARERSIPVETRRIASAGEAQSAPTPYTSYSLFYNGHFITHEIL
TRSKFEKNIDGWIEKYGLPSAE
>lcl|NZ_CP026520.1_prot_WP_042226810.1_3678 [locus_tag=PC41400_RS18630] [protein=hypothetical protein] [protein_id=WP_042226810.1] [location=4107058..4107264] [gbkey=CDS]
MSLIRNVFLVLIVVSMLLREEIALLMLYVKEETTYFGLSLSNAALWSHVLALAAILSLFGYLKSKPRQ
>lcl|NZ_CP026520.1_prot_WP_042226812.1_3679 [locus_tag=PC41400_RS18635] [protein=AraC family transcriptional regulator] [protein_id=WP_042226812.1] [location=complement(4107287..4108936)] [gbkey=CDS]
MNDNQDRPFARRASFYELTGISRETASSGWKQEAAGADSYYVMAVTGGSGGMALNGESYAAGRGKCYLAAPGSGACIHSA
AADLSFYLLKFEVLARQTLGEKAASGLADAADRRGEVPSRMPGAEQENLLEPGEIVCLSFASCVAMLEALYERRRPAGEF
ESFDTYVRFQEFLRFLLQQRAADSGGLDPMQAVERSIAVIRDNYRSTLTVERLASAAGLDRWKYSRLFKERTGTTPLDFL
NRTRIERTKRLLVLTDDPLSGIAGDAGFNNEYYFNRRFKQTVGITPGQYRRNHREQVRVFAPCLEDFLLALEITPVMQWF
SEGWGKQDYLGMDDVPVFDVSDGSLERLTKEKPEFILLDSGTHLPGYSRLAPTYRMDHPGEDWRSTLGITADLLGKKSRV
RDIIGEYESRADKARQALERSVRDQTVAFLRISAEAVILYGGPEQGYTGPVLYGDLGLAPHRLVSQLTGRSRRSVVLTPE
WLAKLDADHLFVTFDRRWGYTAGAREDERIALLPGVRNNSVYEVDFLTWMNYGILSHGKKIDDVLRVLA
>lcl|NZ_CP026520.1_prot_WP_042226814.1_3680 [locus_tag=PC41400_RS18640] [protein=ABC transporter substrate-binding protein] [protein_id=WP_042226814.1] [location=complement(4108956..4109939)] [gbkey=CDS]
MREKSSKKAVWNLMGVLVLAASITGCGGQGERAEEAAVHSTPAGEKSTGDKRIVKDAFGEVEIPAKPKRVAAIYMEDYVS
ALGVTPVVQWYHPNWGIQEYLKLNVPKFDMTGSIEALLDASPDLIIVDGGADKARYDMYSKVAPTYRLPDEMLNDASAIL
KTTADLLGMPDKAESILKEYRKKVDDAKTALEKAVGTEKVAVIRLNVGEKTFTLFGINNRFIGSILYRDLGLTPPPMVKE
MKEFQQIISEEMIPQIDADHIILLPSNGTWASEENKDAVQLLKSPFWKSVPAVKKGHVYQVERTHWQTGAFLANGMKIDD
LLKLLVK
>lcl|NZ_CP026520.1_prot_WP_042226816.1_3681 [locus_tag=PC41400_RS18645] [protein=carbonic anhydrase] [protein_id=WP_042226816.1] [location=4110221..4110862] [gbkey=CDS]
MSNIREILAHNRIFVQKKQALPFQATRYLETKIVILTCMDTQIPEFLIKALGIRNEDATVIQNAGAVVSHPSGSILRSIL
LAICKLKAKEVYVIGHHECGLANFRSAEDSESVRQFVDSDPYMQTLTDAGIDLDGWLTGISSIEESIANSVSMVRSHPLF
PESVLVHGLVIHPVTGRLDVVVEGKHIQPKEGVSALLNAEAAPLDATLDRGNE
>lcl|NZ_CP026520.1_prot_WP_042226818.1_3682 [locus_tag=PC41400_RS18650] [protein=DedA family protein] [protein_id=WP_042226818.1] [location=complement(4110989..4111639)] [gbkey=CDS]
MEQWITETMESFGYFGIFLLIALENLFPPIPSEVILTLGGFMTTTSALTAVGVILAATAGSVAGAVCLYGVGKLMSMERL
QTLVARHGKILRLKPEDIGKAGSWFDKYGVWAVFLGRLVPLVRSLISIPAGSAGMRFVPFLLLTTAGSLIWNSVLVYVGA
AVGASWETIVGYMDIYSNVIYSGLALTCALIVFLYVRRRIVKSEREKKSFEIDPDK
>lcl|NZ_CP026520.1_prot_WP_042226820.1_3683 [locus_tag=PC41400_RS18655] [protein=TerC family protein] [protein_id=WP_042226820.1] [location=complement(4111697..4112467)] [gbkey=CDS]
MDASLILEYGWVLLVLVGLEGLLAADNALVLAIMVKHLPDEERKRALFYGLAGAFVFRFLSLFIISFLVDVWQVQAIGAV
YLIFIALNHIFRKVLVKKGEKGIKKEKEVKAGSSFWATVFKVELADIAFAVDSILAAVALAVALPESGLPRIGGMDGGQF
FIVFAGGLIGLIIMRFAASYFVNLLNRRPGLEIAAFVIVGWVGVKLIVYTLSHPALSVLPEGFAKLPAWKYTFYAVLVLI
ALCGWFLSKEKKETAV
>lcl|NZ_CP026520.1_prot_WP_042226823.1_3684 [gene=flgB] [locus_tag=PC41400_RS18660] [protein=flagellar basal body rod protein FlgB] [protein_id=WP_042226823.1] [location=complement(4112519..4112920)] [gbkey=CDS]
MLVSSTTRMNEKALDIAAARHQAILGNLANSDTPVYKSRFVSFEEELEKATEGKMPLEGITSHKRHFKISGSGEAGLIKI
AVLGPYVFNTNGNSVDMDYEMSLMAKNQLAYSVFAESVSGHYKKMNGLLRDLK
>lcl|NZ_CP026520.1_prot_WP_042226824.1_3685 [locus_tag=PC41400_RS18665] [protein=EamA family transporter] [protein_id=WP_042226824.1] [location=complement(4113432..4114349)] [gbkey=CDS]
MIPFNYVLMCLIFGTTFLAIKVGIDAGVSPFFSAGVRFFLAGAILFIVMVIQKKARLSLLLRKEMMLTGLGLTFGVFASL
YWAEQYVSSGIAAVLSATAPLMIMLLQTGITRQKMSASSWAGCLVGFAGVVILLVPGMTVAFSGIWAAGCVAVLLGQVFY
SAGTVYSRRVIRRFSDTSPIALNAAQMLYGGLMLLLLSLFTEQVHMEAMLVPEAVFSLLYLIVFGSMTGHTIFYWLVAKT
NPVFPSTWLYISPLIALSLGVVLYGEPLSLLSLAGGVTIIAGILLINLKSLRQLAGARSHLVERS
>lcl|NZ_CP026520.1_prot_WP_042226825.1_3686 [locus_tag=PC41400_RS18670] [protein=PLP-dependent aminotransferase family protein] [protein_id=WP_042226825.1] [location=4114474..4115898] [gbkey=CDS]
MERENRSNVHNDKLFEQIYDYLLERIRRGEWKAHDKLPSIRSLAADLRVHRLTVFKAYQLLKRNGDVYVKDKSGYYVHPG
SQSMLGPLNDPIVSAYVHESHLSEIHQVQADYQFSKALLDPNLLPNLYLSEYVKKVFDLYPKVLGTYSTVQGDQELREAL
CGYFTKGFHFFVSPEELLITSGSQEAIDLVARVLVRPRDAIMLERPTYSPAIDVFSRQGAEIIPVDIYPSGYDLEQVERI
MQTSRPRLFYLNPTFHNPTGFTVPAEQRKKLVELAETYKCLLIEDDPGRDIYFGDEPPLPLFSFDTAGYVIYLRSFSKYI
APGLSIAAVACRPSFMNYLIRTKSLSDSGAPLLNQKIFLHYFFSERMQQHLAKLRIALAVRKDIMEEELSGTGWQWTSPA
GGFNLWVELPASVPMDRLLSRCMEQSLTFVPGTICDPLRESTSRVRMSYSYLNETELRAGLRRFKEVVFSLTTF
>lcl|NZ_CP026520.1_prot_WP_156972783.1_3687 [locus_tag=PC41400_RS29885] [protein=hypothetical protein] [protein_id=WP_156972783.1] [location=4116205..4116363] [gbkey=CDS]
MKTRTQRIMLTTPFTLAVLMIAIKIDTILYILVGTFIVGPVCAWVGRGTKEE
>lcl|NZ_CP026520.1_prot_WP_042226827.1_3688 [locus_tag=PC41400_RS18675] [protein=cytochrome P450] [protein_id=WP_042226827.1] [location=complement(4116580..4117809)] [gbkey=CDS]
MEKMTGDDLNSPETMRDIISFYKRLAAQKESLIRLDDFYGMGGAWIAFRHDDVAAMLKDPRLVKDMRKYVPQNEPEAAEA
STYVGKLFEWLRNMPNMLSADPPDHTRLRRLASKAFTPQMIEGLRPRIQQIADELLDTVQERGRMDLVADFAYPLPITVI
SEMLGIPAADRNLFRGWTQKLLHAAVDPRQGSAVEEALEEFIGYIRTLLDDKRRNPGTDVTSGLVQAYEQGDKLSENELL
STIWLLIIAGHETTVNLIGNGMLALLRHPGQMRLLRETPSLLPGAVEEMLRMEGPIVIASRFAGEDIAMHGKVIRQGEIV
LLSLAAANVDPHKFSDADTFDITREENEHLAFGKGIHHCLGAPLARLEGQIAFGTLLRRLPDLRLAVEPERLVYNYGKLR
SLTGLPVLF
>lcl|NZ_CP026520.1_prot_WP_042226830.1_3689 [locus_tag=PC41400_RS18680] [protein=TetR/AcrR family transcriptional regulator] [protein_id=WP_042226830.1] [location=complement(4117889..4118509)] [gbkey=CDS]
MPIQLYARKEILEACLSVFARHGYKDTSTGMLAEAAGISKALIFHHFKSKKTLYLSLLDHCFEKAREELRVDTVPDYEDF
FEAIDTYSRKKLDYFRKHPDEIKLIYEVYYSTPEELKEDIAKKYGHVFASKYEVWERLFDKVPLRNGVDRKNAFELIVIT
LEHFENKFVAEVTDLSAIDDEYIENMFTKMNEFCAMIRQGISQDHT
>lcl|NZ_CP026520.1_prot_WP_042226833.1_3690 [locus_tag=PC41400_RS18685] [protein=NAD(P)/FAD-dependent oxidoreductase] [protein_id=WP_042226833.1] [location=complement(4118779..4119693)] [gbkey=CDS]
MEQQIDVIIIGGGPAGLNAALVLGRARKSVVIIDEERPRNRVTRESHGFLTRDGISPAEFRRIAREQIAAYPSVRFVTDS
AVTVTGADGNFQITTGQGTVYTSKKLLFAAGKKDLPLDISGLADVYGKSAFVCPYCDGWELRDQPLVLIVKGGAALHSAR
TVSGWSKRLTICTNGPDELTEEQREELKARHIPVFDAPIARIESTGGMVEQVVLEDGSAIPCKGIFFAPKLAAGSDLPRE
IGCRMTEGGSVVADPFGKTNVPGVFSAGDAASEMYQAITAASLGSLAAVGINSELLAETWESRI
>lcl|NZ_CP026520.1_prot_WP_042226835.1_3691 [locus_tag=PC41400_RS18690] [protein=Rrf2 family transcriptional regulator] [protein_id=WP_042226835.1] [location=complement(4119743..4120234)] [gbkey=CDS]
MQFSKSTGYALHALIHLAHTERGRNVGIKELSAWLGVSESYLSKIMSKLRQDGIVRAVPGASGGYELARETDQITFLDVI
QVIEGRQHMYECFKISEDHHPMFFESEVLEPESSDGTGSRECLVKKVMIGAERHLNDYLRSHTIQSVLDSASAGNGKGIN
KKP
>lcl|NZ_CP026520.1_prot_WP_042226838.1_3692 [locus_tag=PC41400_RS18695] [protein=hypothetical protein] [protein_id=WP_042226838.1] [location=4120468..4120926] [gbkey=CDS]
MRLNKTAFIFVWLIGAAILFYGIGKLELYLNAESIKSAYSIILLFSGIKIMLYMLTGVYSGTLFLTSQLTVHRALFLAVF
ILFFILSFYPLLIFAVPMPMTDSLILHYQEFAFIAGLCLFMSLYKFRGTSAVRPQPAGEPANANDRTAGFRD
>lcl|NZ_CP026520.1_prot_WP_042226840.1_3693 [locus_tag=PC41400_RS18700] [protein=NAD-dependent epimerase/dehydratase family protein] [protein_id=WP_042226840.1] [location=complement(4121034..4121912)] [gbkey=CDS]
MQNVLILGGTRFFGKKLAANLLCGGADVTILTRGNTADSFGPDVKRLHADRTDAAALRQVIGSADYDVVYDNICYTPQDS
EQAVQLFAGRTGRYIVTSSKSVYPFGGPRKKESDFDPYGYPLPSVYPDQPDYAEGKRLVEAVLFRKAPFPAAAVRFPIVL
GADDYTRRLHFHVEHIKQGLPLGIPNRDALLTFIDSDEAASFLGWLGTSALEGPVNACSKGETTPGKIVSLIEKALNKQA
HVKDETDAEHMSPFGVPESWYMDTSKAEAAGYTFRRLDDWLPDLIRKLAGES
>lcl|NZ_CP026520.1_prot_WP_042226842.1_3694 [locus_tag=PC41400_RS18705] [protein=AI-2E family transporter] [protein_id=WP_042226842.1] [location=complement(4122012..4123097)] [gbkey=CDS]
MYLQSRFFRVCLGIISVLLIILLFSKISFVFNPLITMINILIVPFMMAGFFYYLLRPVIQFLTSRKMNKVLSILLVYFVL
SAIVIVFVVGVWPTLQHQVENFVNGVPGLIENFKQQIGRLQQNRMFSMFAGSESDLSTKLSEYLNRGITAASDYVSNVIS
ILTNFVIIIATVPIILYYMLKESEHIPSSILTVVPKRYRRDGKEVLANIDSALSGFIVGRVIITSLLGVMLYIGFLIIDL
PYSLLLAIAATILNIIPYIGPILGAIPTIIVAFTVSPAMVAWVLVVTVIAQQIEGNLLSPHIYGKRLDIHPLTTVILLLV
AGDIAGILGVILAIPAYIVLKIIIVRIYSLFLAEKVEELVE
>lcl|NZ_CP026520.1_prot_WP_042226845.1_3695 [locus_tag=PC41400_RS18710] [protein=LysR family transcriptional regulator] [protein_id=WP_042226845.1] [location=4123291..4124199] [gbkey=CDS]
MDLKKLHYFIAVAEELHFNRAAKKLNMTQPPLSQQIQVLEEELGVKLFERTKRQVRLTPAGTVFLEESRNLVAQLEQSIK
KTRLAGQGKIGNLSIAFVDSAVGGTMVDVLKAFRERFPDVQLTLTEMTTTGQLQALHEAKIDVGFLRSADPSPHVTSRLY
TNESLLAALPVTHRLAKQEFLSVHSLADEPFILFPRDKGASFHDLILYFCGKHGFQPRIVQEAVQMYTIVNLVAAGIGIS
IVPSSVSAFRREDVVFRSFSEVAPSVPLYASWKTDRNETVLMNFLDVLEEKTAGEVSRISHC
>lcl|NZ_CP026520.1_prot_WP_042226848.1_3696 [locus_tag=PC41400_RS18715] [protein=aldo/keto reductase family protein] [protein_id=WP_042226848.1] [location=complement(4124196..4125185)] [gbkey=CDS]
MKYRKLGKSGLSVSEMALGNWITHGAQVDDRTAQACVNAALDAGITTFDTADVYSETRAETVLGTSLKGIRRDSIELCTK
VFHPTGSGPNDRGLSRKHIMEGCDASLRRLQTDYIDVYYAHRFDSDVSLEETFIAFSDLVRQGKVLYIGVSEWTAEQIKR
GAALARELHVPFAASQPQYSMLWRVIEGEVIPACEREGIGQVVWSPMAQGILSGKYAPGEPVPEGSRASAEAGAPFFDRL
AGQWLNNEVLQAVGRIRPLAEESGLTMPQLALAWVLQNPQVSSAIIGASRPEQVQENVKAAGVRLDAEVMRQIDLILSGI
AEFDPSKTG
>lcl|NZ_CP026520.1_prot_WP_042226850.1_3697 [locus_tag=PC41400_RS18720] [protein=TetR/AcrR family transcriptional regulator C-terminal domain-containing protein] [protein_id=WP_042226850.1] [location=complement(4125449..4126165)] [gbkey=CDS]
MARKRTVSVHPPTGRNEDENLHTPLDRNRIVQTALQLLNESGLKELSMRKIADKLQVKTASLYYHVKDKEQLMQLLLDQI
CETMAWPDSSLPWNKQLISWAEQFKHVLLSHRDAVELFNSGLGGGYERLTQMEKLYGLFVTAGFKDDQIPWIASIVKNYV
LGFAAEEARIYAFAGGVQTTPEELGEQYSRFFRQLPEERFPHLIRLASHTTNTNWAEEFEFGLGVLINGFAAKLQDQS
>lcl|NZ_CP026520.1_prot_WP_042226852.1_3698 [locus_tag=PC41400_RS18725] [protein=VOC family protein] [protein_id=WP_042226852.1] [location=4126361..4126729] [gbkey=CDS]
MKLNHLNLCVNDLTEARLFFEQIFDFQLLGQKGDAIAAMSDGEGFSLVLSSFSKPGAQRPAYPKDFHLGFYRDTPEEVDA
FYNRLLSFGIPVEQGPRKIRNGYTLYFTALDGILFEVTSFAG
>lcl|NZ_CP026520.1_prot_WP_042226855.1_3699 [locus_tag=PC41400_RS18730] [protein=NupC/NupG family nucleoside CNT transporter] [protein_id=WP_042226855.1] [location=complement(4126789..4127970)] [gbkey=CDS]
MKYVLALIGLIVILSLAWLVSSDKKKIKYRPVLVMIALQIILGLILLKTGVGEFLVKGFADGFAKLLGFANVGTDFVFGG
VVNEGIHSFFFHVLMPIVFMSVLIGILQYYRILPFIIKYIGLALSKVNGMGKLESYNAVASAILGQNEVFISIKKQIGLL
PRNRLYTLCASAMSTVSMSIVGSYMTMLKPEYVVAALVLNLLGGFIIASIINPYEVKPEEDILEARDESKQTFFEMLGEY
IMDGFKVVVIVGAMLVGFVALIAMINGIFSSVFGISFQNILGYVLAPFAFLMGVPWHEAVKAGSVMATKIVANEFVAMLD
FVKIQNDFSERTRAIVSVFLISFANFGSIGTIVGAVKGLNEKQGNVVAGFGLKLLYGAALVSFLSATIISIVF
>lcl|NZ_CP026520.1_prot_WP_042226858.1_3700 [gene=deoD] [locus_tag=PC41400_RS18735] [protein=purine-nucleoside phosphorylase] [protein_id=WP_042226858.1] [location=complement(4127967..4128680)] [gbkey=CDS]
MSIHLEAKPGEIAERVLLPGDPMRAKFVAEHFLDEVRCHNEVRGMYGYTGLYKGVPVSVQGTGMGNPSMSIYVTELIRDY
GAKKLIRIGTCGAMQPNLKIRDLVLAQAVSTDSNMTDKMFHGCSYAPTADFGMLMQAYQGAKRSGANVFAGNVYNSDEFY
RHTLDRLHTFMEFGVLAVEMESTALYTLAAKYGVKALSILTVGSQLLTNERSAHKDSEQSFNQMAEVALQTIIEDTV
>lcl|NZ_CP026520.1_prot_WP_042226861.1_3701 [locus_tag=PC41400_RS18740] [protein=helix-turn-helix domain-containing protein] [protein_id=WP_042226861.1] [location=complement(4128810..4130249)] [gbkey=CDS]
MNKDFKLMKALIELLDSEERWFTFGEAERELGISDKSVRKLVEEITALLPGTMSIEVSRGKGIFFRRERHGPTIREILVE
QFRQTSYYRLMHRLFVYGGQASAEELAEFLYMSASSFKKFIIQLNREDLKPSCLRLAYSSPALKGNEIHIRYFYWKLFSD
SHPYTGWPFAEIDFGQIQEWITDLEEEQGIVYFLNSKRSLIFLIAVTLERIRQGHSVSVNGKLYDWENGAFYESVRRLAG
KLGQEYGLDIAGEEIYFLQSLCSLSQYHYRNEREMALVQEEALHRNEEPKYRMLNNLLPLLRETFPSLAAGHRFVMEMCE
FFEKLMIDNAVPELMAVSKSRLIAYMRREEPLIYEGVTECINRWSREHPAVSVSSYHLVKLGFIVSSNVRYKSKKAFLII
GEEFSVRHYVANRIKREIGDELTIETSILHELSDETVREREIDFIISTMPVALKTVPAVIISAIPTKRDLENIRKELLE
>lcl|NZ_CP026520.1_prot_WP_042226863.1_3702 [locus_tag=PC41400_RS18745] [protein=ABC transporter ATP-binding protein] [protein_id=WP_042226863.1] [location=complement(4130482..4132215)] [gbkey=CDS]
MKGGGLVSQVKELGYLMRFMNRRRKIQYYTGLGITAVNQTLFLVSFSLVVQNLVNFADYRDSALMYRAFYILAGALFLEN
VISPWFTYLYQRSVEFTIVDVRERLYRKFCRLRPKYLEQTHHGDLMSRVNNDVAAVELTFSSIYFMLLLQVVFSVGSIIS
MVLIDWRFASVSLLVLLLSFSISARFARQIRVLSEKSLQTLARMTEKFKDFIGGIQLVKLFHIRPVYSQYRSLNEDVTRL
AEQTAKKNGLQAAVNHFVSYVTFCGIIVIGSLLYAFGLIGMGSVAALAVLQINLTHAIMNFGTVLSMAQNSLAGAHRIEE
ILSEEEEPERLGLPDPKRSSEAMVEFRDVEFSYQADKKVLYRMSMKVYPGQVAAIVGASGSGKSTLIKLLLGFYPVDSGE
ILLGGKLFGSYTLDEIRGQIAYVPQEPFLFSGTIEENIRYGNPQATTDDVIEAAKAAYAHHFIEELPGGYGTQAGERGAS
LSGGQRQRIAIARAILKNAPILLLDEATSALDNESEHWVQQALNELMKGRTTILIAHRLSTVEKADSITVMNQGTVVERG
THRDLLEQGGYYARLYG
>lcl|NZ_CP026520.1_prot_WP_129112510.1_3703 [locus_tag=PC41400_RS18750] [protein=ABC transporter ATP-binding protein] [protein_id=WP_129112510.1] [location=complement(4132212..4133996)] [gbkey=CDS]
MAAPKSKRVYAAWKAFRWLLTFVSRHRGWMIAGTVSAIAAAIIEVWTGSLIEQLTSQAENGAGPVVIEIVWTVFLVILIG
VPAKYLMIYGVEKSSAFAVRDIRNHVMNHIGKLPIHYVEKQHSGDILSRISSDLQLIHQFMSRDLAQWFYHPLLFVGCFT
YLIWIQWELMMYSLLLFPVSLLVSHWIGKQLQQLTEEAQANMGRMNMNLQDTLGGMPVVKSYLLSGMLFRSYQSLLLLTA
QKKLAVKKRESWVNPLLSTLMLSPIIFAVSYGSYLISKGELGTGELIAFLYLLNLCLEPLDHIPELITHTFEMSGALKRV
SDILEQPAENQEGHAITKTASPPVEFRDVSFSYETGSPILQNLSFTVPEGKTIALVGASGGGKSTVFKLLCGFYPLEDGQ
GNISVFGQQVKGSNPEQLRSHFSVVTQESYLFSGTVADNIAYGRQDASMDEVIAAAKSAQAHAFIMSLPEGYQTYVGERG
GFLSGGQRQRIAIARALLKNAPILLLDEPTSALDPESEGAVQEALQVLMSERTTIVIAHRLSTVQNADEIWVLEQGRLVE
AGTHAQLLEKKGLYAQSYLQEFAGLAESKEVVYT
>lcl|NZ_CP026520.1_prot_WP_042226868.1_3704 [locus_tag=PC41400_RS18755] [protein=non-ribosomal peptide synthetase] [protein_id=WP_042226868.1] [location=complement(4134028..4137276)] [gbkey=CDS]
MKSLFEKEEMYWKEKFNPDDSLTFLPYNPPSKAVGQPNSEEKPDVVSLTLPPMLSERILRLANGSDLAVFMIVLAGVNCL
LHNYARRDNVLVGIPAATGPGDEPSIHDFLLIKTYINRQSTFKSLLGQVKLAIGEALENQHLPFRKMTGSLNIETTPEGT
PVLNTIVTYKKIHKADIQGKAAADTVFQFESNENSIGMNVSFNSGRYNRDFIEKAAGHLVQLMSILLVQPDLEFGDADML
SPNEKQQLLVDFNHTETHYEREQTIHGLIEEQAARVPDATAVVYEQERLTYRELNERANRLARTLRAAGVQPDQLVGLMA
GRSLEIAVGILAVLKAGGAYVPIDPEYPEERIRYLLEDSGAKLLLTQRRLLDRVPFEGTVLTLDDESSYSGDGTNLEPVN
GPEHLAYVIYTSGTTGKPKGVMLEHHGLVSLKNMFRDTLGITETDRVVQFASLSFDASCWEVFKALFFGAALYIPTAETI
LDNRSFQRYMHENGITAAILPPTYAAYLNPDHMPDLQKLITGGSAVSAEFVQQWKDKVRYFNAYGPTEASIVTSVWTACT
GDIGLKSVPIGRPIQNHRIYILNDYRQPVPVGVEGELCIAGVGLARGYLNRPELTAEKFVDNPFSPGERMYRTGDLARWL
PDGNIEYRGRIDHQVKVRGYRIETGEIEEQLLKVSSVQEAIVIAREEADGQQLCAYFVAQSKLTAAALKEALAKELPGYM
IPAHFVQLLRMPLTPNGKIDRKALPAPEESVEQAEEYIAPRTLMEVKIAQIWKDVLGLRRVGVKDNFFELGGSSLSLMRL
IQAVYDETGIEVPLNQQFKNLTVEAMAFGSGETGLEKGQSFIKLNKNGALNVFCFPPGSGFGIGYRELANLLDDRFVLYG
TDFKEEAKSYEEMLDDYIDEIVSIQEEAPYVLLGYCFGGNLMFEVAQRMEKRGYRVSDLIMVDSWIKETLTPAETTESEW
QEILEDLSDTEKEMMANPVIRERVSIKVRATLTYEAQLLNSGTVEGRIHELIAEDSDAFRLEQGLPTWRGAAQDTYTEYK
LAGAHEKLLEPECLDGTAGVILRILEEVAQRNGASSKVLHGS
>lcl|NZ_CP026520.1_prot_WP_129112511.1_3705 [locus_tag=PC41400_RS18760] [protein=non-ribosomal peptide synthetase] [protein_id=WP_129112511.1] [location=complement(4137383..4152310)] [gbkey=CDS]
MKVAFEKEILFWNEKFSGEDDSLTLLPYSKAPSHAAPSVQSVSGTLSDAAAQRVIQMSKGAPLAAYMILLAGVQVLLYKY
TGDSAVLVGMPVVKKPQETRRPINHTVILKNRIGGESTFKSVLNELKVSLPEAIRHQQIPFLKMTEKLDLQYTDGVPVVN
TLVSLNELHGTDVGGNVVCDCLFQFRLEDGSIHLQIGYNENLYHQGFMTGVVDHLDRLLAGGLNNLDQEAAGAAMLSPEE
ERRLIHAFNETETDYESEKTIHGLIEDQAVRVPDATAVVFEQERLTYRELNERANRLAHTLRAKGVQPDQLVGLMAGRSL
ETVVGILAVLKAGGAYVPIDPEYPEERIRYLLEDSGAKLLLTQRRLLDRVPYAGTVLALDDESSYSADGTNPEPVNGPDH
LAYVIYTSGTTGKPKGVMVEHHGLVSLKNMFRDTLGITGSDRVVQFASLSFDASCWEVYKALFFGAALYIPTSDTILDYQ
LFERYMDENGITAAILPPTYAAYLNPDHMTGLQKLITGGSAVSAEFVQQWKDKVRYFNAYGPTEASIVTTVWAAGPTDTG
LKSVPIGRPIHNHKIYIVDKHGQPVPEGVAGELCIAGVGLARGYLNRPELTEEKFVENPFASGERMYRTGDSARWLPDGN
IEYMGRIDHQVKIRGYRIETGEIEEQLLKIPSVQETIVIAREEESGLQLCAYVVGDQSLTVSDLRAALSQNLPGYMIPAH
FVQLPRMPLTPNGKIDRKALPAPQGNALGGGEYVAPRNEAEQALADVWKAVLNTERVGVTDHFFELGGDSIKSIQVSSRL
HQAGYKLEIRDLFKYPTIAQLSLHVKPVARTVDQGEVTGAAVLTPIQQWFFEREFADAHHFNQSVLLHRKDGFDETAIRK
VLQKLTEHHDALRMAFRKTEHGYTAWNRGIGEGEMYSLEITDFRKAANPGAAVEAKANEIQSGIGLEDGPLLKAGLFRCA
DGDHLLIAVHHAVVDGVSWRILMEDIALGYEQAGRGQDIRFPAKTDSYLGWSGQLAAYAQSPDMDMEKAYWQGVERTSVP
PLPKDMDEVDSQQRDSESLFAVLTPDETELLLKRVHRAYNTEMNDILIAALGAAVQKWSGHDRVLVNLEGHGREFIGTDI
DITRTIGWFTSKFPVVLEPESGKDLSHQIKAVKESLRRIPNKGIGYGVCRYLSEKTDGLVWGAEPEINFNYLGQFDDDLQ
NNEMGLSPYPGGSPASDRQARSFVLDINGMVLNGSLSLDLSYSRKQFHKATIERLVSLLEQSLREIIAHCAAKEKTELTP
SDVQFKGLTVTDMERIAEDTAPIGEIENIYTLTPMQKGMWFHSAMDPQTGAYFEQARFTLLGELDVDVFSKSWAELAARH
VVLRTNFYSGAGAEPMQIVYRDKPVGFEYEDLLQLDSAGRTAYTDKKAEEDRLRGFDLERDALVRVTVLRTDEHRYHVLL
SFQHILMDGWCLPQLTQELFETYSAFVRNEKPERTASTDYSQYIEWLEKQDEESASRYWSGYLEGYDGQTELPKKTDAKS
GPFLPEHVVCELGKSLSERMERVAKQQQATLNTLLQSAWGVVLQKINGTGDAVFGSVVSGRPAEIPGIEEMIGLFINTVP
VRVTSEPDTSFAEVMAKVQEQALESGRYDYYPLYEIQARSAQKQDLIHHIIAFENYPMDEQMQQAGGSQTGGLAIADVEM
AEQTNYDFNMTVVPGEAIEIRFDYNAGVFAKADIERLRGQLLLVLEQVVNDPRVKVGDLELATAAEKSQIIEAFNDTTTE
FPRDKTLHQLFEEHAERIPDAVAVVFEQKELTYRELNERANRLARMLRGEGVGRDCLVGIVVERSIEMIVGIMAVLKAGG
AYVPIDPEYPEERIRYLIEDSGARVMLTQRHLQERIPVQRVVIVLDDEGSYHPDGSELERINESADLACVIYTSGTTGNP
KGNLVSHRNIIRIARNTNYVEITEQDRVLQLSSYSFDGSISDIFGALVNGAGLVLIPREIMLEIGQLAELIERQRITVSL
ITTAFFNVLVDVNVDCLRHMRAILFGGERVSVGHVRKAVDHLGPGKIIHLYGPSESTVYATYYVVNEVAEDAVNVPIGRP
ISNTTAYIVDARNNLQPIGVAGELCVAGEGLVRGYLNRPELTAEKFVDNPFVPGERMYRTGDVARWLPDGTIEYVGRMDD
QVKIRGFRIELGEIEAQLLKVESIRKATVIARETASGDKQLCAYFTADGELQASRLRSALSRELPGYMIPSYFVQLEEMP
LTTNGKVDRRALPAPELSLQEGTEHTAPRTPLETDLAAIWQSVLGLEKVGVKDNFFELGGHSLRATTLVSRLRKELHVDF
PLKDVFVYSTVEEMAQAIANLAQLSFSSIPKAEESDYYPLSSAQKRLFILQQLEGADQSYNMPGALLLEGALDRHRFEEA
FRGLIARHETLRTGFGIINGEAVQRIDSDADFSVEHIRASEEEAVEAVQKFIRPFDLAAPPLLRVGLIELAEDRHILMFD
MHHIVSDGVSMDVLVEEFVRLYSGEELEPLHIQYKDYAVWQQSEGQKEQLKRQEAYWLDVFRGELPVLELPTDFARPAVQ
SYEGQTLEFFVDAQKSEALKRLAAESGSTLYMVLLAAYTVLLHKYTAQEDLVVGTSIAGRTHGDVQPLIGMFVNTLAIRN
YPAAGKTFLSYLQEVKETTLGAYEHQTYPFEELVEKVQTARDLSRNPIFDTMFSLQNLENKEFELEGLKLTSYPSEYGMA
KFDLSLDVVEESGGLECSLEYATSLYKKETVERMAQHFEQLLGAIAEKPEAKIAELELLTLFEKEQLFEAFNPAMSELPQ
VQTFHRLFEEQAVSTPDAAAVVYEDKQLTYRELNEKANRLAATLREGGVGRESIVGILAERSVELLVGVIAVWKAGGAYV
PLDPDYPADRIRFMLEDSGATVLLTQTSLRESSEAWLEAEETSLRTILCLDDEKSYSGDGENAAVGVVLEGSVNGDSADS
AASGSLSDSGTPDGVPAETQPHDLAYVIYTSGTTGRPKGVMIEHGSLVNTAAAYRQEYRLDRFPVRLLQLASFSFDVFVG
DIARTLYNGGTMVIVPKDDRIDPARLHGWIQEQQITVFESTPALIVPFMEHVAQNALDMSSMELLITSSDSCSVGDYRVL
QERFGSQFRIINSYGVTEAAIDSSFYDEPLAKLPETGSVPIGKAWLNARFYIVDAHLNPVPVGVLGELCIGGPGVARGYL
NRPDLTEEKFVDSPFVAGERLYRTGDLARWTADGNVDFIGRIDHQAKIRGYRIETGEVESQLLQADGVREAVVLVREDGS
GQKALCAYFTADGELAVNSLRSALAQELPGYMIPSYFIRLERIPLTPNGKVDRKALPAPEAGVDGGAEYIAPRTPLEAKL
AAIWQDVLGLQKEIGIRDNFFDLGGHSLRATTLVGKVHKELNVDLPLRDVFRHSTVEEMAAAIANMERQEHVSIPLAEER
DHYPQSSAQKRLYILNQLEGAELSYNMPGALLLEGALDQNRFEEAFRSLTRRHETLRTGFEMVNGEAVQRVYADVDFSVE
HFSAAEEEAAETVQNFVRPFDLAKPPLLRVGLVELAVERHILLFDMHHIVSDGVSMDVLVEEFVRLYSGESLEPLRIQYK
DYAVWQQSETQKEQLKRQEAYWLDVFGGELPVLELPTDYARPAVQSYEGRTLEYFLEAQKSDALKQLAAETGTTLYMVLF
AAYTVMLHKYTGQEDLIVGTPIAGRNHGDVEPLIGMFINTLAIRSYPAAEKTFLSYLQEVKETTLGAYEHQNYPFEELVE
KVQVARDLSRNPIFDTMFTLQNTDNKDFELEGLRLTSYPSESTMAKFDLSLDVMEDSGGLDYSFEYSTALYRKETVEMMA
QHFEQLLGAIAEKPEAKIAELELLTLSEKEQLFEAFNPAMSELPQVQTFHRLFEEQAVSTPDAAAVVYTDKQLSYRELNE
KANRLAARLRADGVGRESIVGILADRSVELLVGVIAVWKAGGAYVPLDPDYPADRIRFMLEDSGATVLLTETSLREDAEA
WLEAEETALRTILCLDDERSYSGDGENAAVGSKAVGNANESVESARVSAGQGDGAALSSTADSESDGIPYETQPHDLAYV
IYTSGTTGRPKGVMIEHRSLVNTAAAYRREYRLDRFPVRLLQLASFSFDVFVGDIARTLYNGGTMVIVPKDDRIDPARLH
GWIHEQQITVFESTPALIVPFMEHVAQHELDMSSMELLITSSDSCSVGDYRVLQERFGSQFRIINSYGVTEAAIDSSFYD
EPLAKLPDTGSVPIGKAWLNARFYIVDAHLNPVPVGVLGELCIGGPGVARGYLNRPDLTEEKFVDSPFVTGERLYRTGDL
ARWTAEGNVDFIGRIDHQAKIRGYRIETGEVESQLLQADGIREAVVLVREDGGGQKALCAYFTADGELAVSSLRSALAQE
LPGYMIPSYFIRLDRIPLTPNGKVDRKALPAPEAGVDGGAEYIAPRTPLEEKLTAIWQDVLGLQKEIGIRDNFFDLGGHS
LRATTLVGKVHKELNVDLPLRDVFRHSTVEEMAAAIARMERQEHVSIPLAEEREYYPLSSAQKRLFIQHTLEGADQLYNM
PEMLQLEGPVDIDRIQEAFRTLIARHETLRTGFEMVHGEPVQRVHAQVEFAVETSKSAKEDAAQIARSFVRAFDLRTPPL
LRAGLIELEPELHILMFDMHHMVSDGVSMSILVDEFSRLYAGEELPPLRIQYKDYAAWQQSETYKERTRQQESYWLEALA
GDLPDVELPADYDRLPVRSYEGAHLEFEVGADLTAELGVLAARSGSTMFMILLSAYNVLLSKYSGQEDVIVGTPVAGRTD
ADLEPVIGMFVNTLAIRNRPAGDKTFLSFLEEVKESALGAFEHQDYPFEELVERLNVKRDAGRNPLFDTVFDMQNIEEHE
ASLGTLRLKPYELDDLEEAKFDLTLFMSEDSGTLSGGFFYGTKLFKEARIRTMVQDYLHILSQICENPEKRLSEIQCLKP
SSGTKSSVEAIEFAF
>lcl|NZ_CP026520.1_prot_WP_053228637.1_3706 [locus_tag=PC41400_RS18765] [protein=non-ribosomal peptide synthetase] [protein_id=WP_053228637.1] [location=complement(4152394..4164831)] [gbkey=CDS]
MRENTNKRFALTQAQRRIWSMEMMYPNTSVGIIVGTLFIRGPVNTDALMQAIYKLVRDHDAFRIRISSSDAGEPVQWFAN
ESDIEPRCDYLEMGSEAEAKEWLERFNHTPLSIFDEQLYQFCLFNLNNEEHWFNLKIHHSIADGIASHLICTTIMENYTK
LIDGQEPDAKAKGTYLDYIYAEQEYERSDRYQKDRAYWLDKFESFSEITRMKPSSPYSVSTEADRVSVPFAGKRYEQLQR
FSEEHNISTFTVFLAALYILLHKVTGNEDITVGTIYANRTSKQEKETLGMFVSTVATRMLLEPDQHVLAFLQHVSKEQKA
NLRHQKYPYNQLIQDLRERNKQWDVPELFRVDIEYLPVNWTYYDGLSVIRKSSHSGHEVGDFAVHVVDMLDDKEIDLNID
YRVQLFDESEILRISEQLFTIIDQIVLNPLQTVRELSMLSDGEKNKILTEFNPPASELPPVQTFHQLFEEQTVLNPSATA
VVYENNELTYGELNERANRLAGSLRGAGIGRESVVGILADRSVDLLVGVMAVWKAGGAYVPLDPDYPSDRIRYMLEDSGA
AVLLTQSGLRDRAGTWQGEGGALRTILCLDDEKSYSEERENAPAESRPDDLAYVIYTSGTTGRPKGVMIEHRSLVNTAAA
YRKEYRLDQFPVRLLQLASFSFDVFVGDIARTLYNGGTMVICPKDDRIDPTRLHGWIQSRQITVFESTPALIIPFMEHVA
QQKLDMSSMELLITSSDSCSVEDYRLLQERFGSQFRIINSYGVTEAAIDSSFYDEPLYKLPEAGNVPIGKAWLNAKFYIV
DAQLNPVPVGILGELCIGGPGVARGYMNRPELTEEKFADSPFAAGERLYRTGDLARWMEDGNVDFIGRIDYQVKIRGYRI
ELGEIETAMLRFDGIRQAVVTDRTDERGQKYLCGYVVSESEIDTRELHAYLEQTLPSHMVPARVMRLDRLPLTPNGKVDR
KALPEPEGSVQAEVKYIAPRTPAEQALAAVWQSVLGVDRVGLSDNFFGLGGDSIKALQVSSRLLQAGYKLVMRDLFHYPT
IASLSPQLQTAGKTASQEEVTGAVTLTPIQRWFFGQNPADPHHSNQAFMQYRREGFEETALRKTIAKLVEHHDALRTVYR
KTEAGYEAWNRGTEEELFSLEVADFTGLGNCKAAVEAKASEIQGSIDLENGPLVKLGLFRCPDGDHLLMAIHHLVVDGVS
WRILFEDFAAGYEQALGGQAISLPHKTDSFQTWAGQLSRYADSEALESERAYWRQIGQLEQTLLPKDLGQAKSTLKDSDV
VTVRWTEGETEQLLKQAHRAYNTEMNDLLLTALGMAVQAWTGQNRVLVNLEGHGREDILPEVDITRTVGWFTSQFPVLLE
SGRDQTLSQQIKQTKEGLRRIPNKGIGYGILRYLSEGREGDLFDSDPEISFNYLGQFDQDYESSGLEPSPFSTGNDISEN
AMMDFALDMNGMVTEGSLELTIRYGTTQYRRETIERLGSLLETSLRAVIAHCLSKDLPELTPSDVLLPELSVEELEQLTA
HTAAIGELENVYTLTPMQKGMLFHSLLDADSEAYFEQVTFDLQGSLDAEAFAKGLDVLIQRHEALRTNFISGWKDEPVQI
VFRQRKGEVYYEDVSGLDEAEREQALAELAEADKARKFDLAHDSLMRVTLVRTDEQTWHVIWSHHHILMDGWCMSFMIKE
VFESYFAFLEKRKPELPPVTPFARYIEWLEKQDRNAAADYWSGYLNGYDQHTKLPQEKAQRKGERFNAVELEFEFSRELS
ERIERVSRQNQVTLSTFVQTLWGLVLQVYNNSRDVVFGCVVSGRPADIAGVESMIGLFINTIPVRVRSEAGESVADVLRK
TQEQALASGAYDTFPLYKIQALSEQKRDLISHIMVFENYPMEERIEQLIGGNGALAISNIQAPEQTNYDFDITVIPGERT
VIRFTYNADTFSEEDIRGIHGHFTRAVEQAAADPNIPVEQLELLTLSEKEQLFEAFNPAMSELPQVQKFHRLFEAQAVST
PDATAVVYEDKQLSYRELNEKANRLAATLREGGVGRESIVGILAERSVELLVGVIAVWKAGGAYVPLDPDYPADRIRFML
EDSGATVLLTQTSLRESSEAWLEAEETALRTILCLDDEKSYSGDGENAAVGVVLEGSVKGDSADSAASGTPDGVPSETQP
YDLAYVIYTSGTTGRPKGVMIEHRSLVNTAAAYRREYRLDRFPVRLLQLASFSFDVFVGDIARTLYNGGTMVIVPKDDRI
DPARLHGWIQDKQITVFESTPALIVPFMEHVAQHELDMSSMELLITSSDSCSVGDYRVLQERFGSQFRIINSYGVTEAAI
DSSFYDEPLAKLPETGSVPIGKAWLNARFYIVDAHLNPVPVGVLGELCIGGPGVARGYLNRPDLTEEKFVDSTFVTGERL
YRTGDLARWTAEGNVDFIGRIDHQAKIRGYRIETGEVESQLLQADGVREAVVLVREDGSGQKALCAYFTADGELAVSSLR
SALAQELPGYMIPSYFIRLERIPLTPNGKVDRKALPAPEAGVDGGAEYIAPRTPLEAKLAAIWQDVLGLQKEIGIRDNFF
DLGGHSLRATTLVGKVHKELNVDLPLRDVFRHSTVEDMAEAIANMERQEHVSIPAVEKREFYPLSSAQKRLYILNQLEGA
ELSYNMPGTLILEGALDRNRFEEASRGLIRRHETLRTGFEMANGEAVQRVYPDVDFSVEHVVASEEEAAETVRNFVRPFD
LAKPPLLRVGLVELAVERHILMFDMHHIVSDGVSMDVLVEEFVRLYSGEALEPLRIQYKDYAVWQQSEGQKEQLTRQEAY
WLNVFRGELPVLELPTDYARPALQSYEGRTLQFFMDAQKSEGLKRLAAETGTTLYMVLFAAYTVMLHKYTAQEDLIVGTP
IAGRTHGDVQPLIGMFVNTLAIRNYPAADKTFLSYLQEVKETTLGAYEHQNYPFEELVDKVQVARDLSRNPIFDTMFTLQ
NTENKELQLPGLRLAPYASEEIVSKFDLSLDVTETSGGLDYLFEYATSLYKKETIERMAKHYEQLLNAIVKSPEVKIADL
ELVTPSEKEQLLEAFNPAMSELPEVQTFHRLFEAQAISTPDATAVVYEDKQLSYRELNEKANRLAATLREGGVGRELIVG
ILAERSVELLVGVIAVWKAGGAYVPLDPDYPADRIRFMLEDSGATVLLTQMSLRESSEAWLEAEETALRTILCLDDEKSY
SGDGENAAVGVVLEGSANGDSADSAASGSLSDSGTLDGVPSETQPHDLAYVIYTSGTTGRPKGVMIEHGSLVNTAAAYRR
EYRLDRFPVRLLQLASFSFDVFVGDIARTLYNGGTMVIVPKDDRIDPARLHGWIHEQQITVFESTPALIVPFMEHVAQNA
LDMSSMELLITSSDSCSVGDYRVLQERFGSQFRIINSYGVTEAAIDSSFYDEPLAKLPDTGSVPIGKAWLNARFYIVDAH
LNPVPVGVLGELCIGGPGVARGYLNRPDLTEEKFVDSPFVTGERLYRTGDLARWTAEGNVDFIGRIDHQAKIRGYRIETG
EVESQLLQADGVREAVVLVREDGGGQKALCAYFTADGEHAVSSLRSALAQELPGYMIPSYFIRLERIPLTPNGKVDRKAL
PAPEAGVDGGAEYIAPRTPLEAKLAVIWQDVLGLQKEIGIRDNFFDLGGHSLRATTLVGKVHKELNVDLPLRDVFRHSTI
EEMAQTIAGLEHREHTAIPVLEKRAHYPLSSVQKRLYIQHQMEGAELSYNMSGLMVLKGELDRGRFEAALRGLIGRHEIL
RTVFEMVDGEPAQVIYPEVNFALEYRQSDEEGAREIAAQFVRVFDLEQPPLLRVGLIGLERNRHILMFDIHHLVSDGVST
GILIDDLLRLYTGESLQPLRIQYKDFASWQQSEAQSERLKKQEAYWLDVLDGELPRLELATDFARPAVRSYEGDVLDFAI
DKQQSEALRQIADENGATLYMVFLAAYTILLHKYSGQEDIIVGTPVSGRTHADTEPLIGMFVNTLAIRNYPADEKMFLSY
LEEIKETMLGAYENQDYPFEELVEKVQAARDLSRHPIFDTVFVMETREDRISRFGELTVEPYIEAQTVAKFDLTLEVTHE
EEGMSGHFEYATKLFTRNMIDNFAEDFLTILSKICEQPHVLLKDLSLNGSSEQEEDLLEAIDIVF
>lcl|NZ_CP026520.1_prot_WP_042226871.1_3707 [locus_tag=PC41400_RS18770] [protein=hypothetical protein] [protein_id=WP_042226871.1] [location=4165666..4166529] [gbkey=CDS]
MFKRLLRTTALVLTVGSTLLTGTVIQSAPAQAASNMVPDYEVKILMSPSTVLGSDNKLKSDVLDTFAMPSSVTKMNVAFL
DTNAKNIYNGGWSPRIRKTEGESKFELSYKKRYPIVNNDINAALTKANQDGFDSSDTNYEAQVEWGYQNKTLSITRSKNA
SKSGYSGMDLPSVSDARSLLISNAPGKFDNWLSNGWGTSLLGSSRYYGPVLAKRSIGTWNGLKLYIEVWPIRNSAGTGTD
YIVEASFKTTSESTASAKHDELIALLQSKGWFLAQDSLKTQLIMDRY
>lcl|NZ_CP026520.1_prot_WP_042226872.1_3708 [locus_tag=PC41400_RS18775] [protein=nuclear transport factor 2 family protein] [protein_id=WP_042226872.1] [location=complement(4166777..4167160)] [gbkey=CDS]
MENRAAISLVEKYVEAYNAFDIEGMLLLMQEKIVFRNISNGVVNAETEGKDAFRELAGQSKKLFSLRCQTITDYTVKGDR
VEVEIDYEGTLAADLPHGPKAGEKFALKGKSVFTIEDGKLLVIEDYS
>lcl|NZ_CP026520.1_prot_WP_042226873.1_3709 [locus_tag=PC41400_RS18780] [protein=TetR/AcrR family transcriptional regulator] [protein_id=WP_042226873.1] [location=complement(4167182..4167772)] [gbkey=CDS]
MPRSKEENDRVRQLAQENIRTSAMQVFIEKGYYNASIADIAKRAGVSKGLLYNYYKGKEELLGDMVQSRIEEIEDVMQKA
AALGTPSEQLKHIIDGAIDNVLHHPKIYRFYLHLQTQPEEDLLLSRYSQMLNEEMARQFQLQCEMFRKLGVPDPEIRSLH
FSSAIHGISLMISTYPEHYPVRQVKEQLFRDFCVTE
>lcl|NZ_CP026520.1_prot_WP_042226874.1_3710 [locus_tag=PC41400_RS18785] [protein=alpha/beta hydrolase] [protein_id=WP_042226874.1] [location=complement(4167772..4168530)] [gbkey=CDS]
MDLFYEVKGEGTPLVFIHSPGVDSREWKFIAPILAQSFQVITFDGRGTGKSPAPTAPANLVEDLKRLLHHLNIEQAVLVG
HSMGGQAAADFALTYPSMVSRLILLSPSLSGFPYSKEFTEWIAQVNSLAPDLDKMIDTSLAGPNYRVVMSSPQRDFLIEM
HTDYMRKVFTEWKSFEVNWPQPPAMERLEEISAPTLFLQGTVEWGDMERVAEEYKRVPSCKFIRIEGADHMLTLTHPAET
AGYIEQFLKELV
>lcl|NZ_CP026520.1_prot_WP_042226875.1_3711 [locus_tag=PC41400_RS18790] [protein=GNAT family N-acetyltransferase] [protein_id=WP_042226875.1] [location=complement(4168686..4169138)] [gbkey=CDS]
MEIRTAQRGDLEEIVRLLADDELGSHRERYEKPLPPAYEEAFSAITGQTGNSILVAEIGGEVVGCLQLTLIPGLARMGMT
RAQVEGVRVDSRYRGHGIGKALLEHAIEIAKMHGCGLVQLTTDKTRGDAHRFYGKLGFTASHEGMKLSLD
>lcl|NZ_CP026520.1_prot_WP_042226877.1_3712 [locus_tag=PC41400_RS18795] [protein=excinuclease ABC subunit UvrA] [protein_id=WP_042226877.1] [location=complement(4169341..4171590)] [gbkey=CDS]
MRPEYIVLRGARENNLKDIHLRIPKRKITVFTGVSGSGKSSVVFDTISAEAQRQLNEMFSTFIRNRLPKTGQPEVDSIEN
LSTAIVIDQKRLGGNSRSTVGTATDINALLRLLFSRVGTPFIGYSNAFSFNDPAGMCPGCQGMGTRAVPDIRKLFDRSKS
LNEGAVRFSTFTAGTWYAKHHLLTGLFDPDKKLRDYTDVEWQRLLYGKSENLQLPSDGGMPSKRFEGVVDRFIRLYFKKD
GPSLPDNVKRYITFSPCELCGGTRLNEAALGCKINGQNIAEAAAQEIEDLLPFLKSLLGDKTAAPILSGMIPPLQHLLDV
GLGYLRLDRETKTLSGGESQRIKMVRHLNSSLTDMLYIFDEPSTGLHPRDVERLNGLLCKLRDKGNTILVVEHDRDVIRI
GDYIVDMGPEAGAEGGRIVYEGNLEGLYEADTLTGKNMKREIPFNPHPRKPSGWLTIENASLHNLKNITVRIPSGVLTVV
TGMAGSGKSTLIHDVFAAENPEAVVIDQSPVSGNIRSTPATYTGIMDEIRHLFARTNGVSDSLFSFNSKGACPDCGGLGF
LETDMAFLEPVKTLCERCGGKRFKEDVLGYTYKGKSIYEVLKMTAHEALHFFEEEEIVRPLQAIRDVGLGYLALGQPLST
LSGGECQRIKLASKLHRQGRIYILDEPTTGLHMADVSKLLAMMNRLVDKGNTVIVIEHNGDVIKQADWIIELGPEAGKNG
GEVIFEGTPLELAEADGSVTGPFLRDTRV
>lcl|NZ_CP026520.1_prot_WP_042226878.1_3713 [locus_tag=PC41400_RS18800] [protein=aminoglycoside 6-adenylyltransferase] [protein_id=WP_042226878.1] [location=complement(4171700..4172566)] [gbkey=CDS]
MRSEQEMFDLIIKVAENDSRVRAVGLNGSRTNPNVSKDIFRDYDIVYLVTDVESFRNDLDWVNVFGERMIMQTPDDMALF
PNEPGNRFAYLMLFTDGNRIDLTLIPLEEKEMYCREDKLTVILLDKDNDLPALPPPTDEDYWVQRPDAQRFADCCNEFWW
VSTYVAKGLWRQEILYAIDHVNLVRAMLLKMLEWKVGMDTGFSLSIGKNSKFLKRYLDEESWNKLLSTYETGDYEQVWNA
LYVTGELFEPVALQVAAEFGYTYPAEEAEKIKPYLRRVQLLAPDARDI
>lcl|NZ_CP026520.1_prot_WP_042226881.1_3714 [locus_tag=PC41400_RS18805] [protein=FAD-dependent oxidoreductase] [protein_id=WP_042226881.1] [location=complement(4173101..4174531)] [gbkey=CDS]
MTFKVAVIGCTHAGTAAVKNIAAQYPDAKITVYERNDNISFLSCGIALHVGGVIGDAQGLFYSSPEELARLGVVTRMRHE
VLEVDVQAKTLRARNLESGEEFGDTYDKLVVTTGSWPVIPRMEGIGLENIQLCKNYGHAREIIARAADAKRITVIGAGYI
GIELVEAFRDLGKDVTLIDNMDRVMAKYLDPEFTALADNELRERGVQLALGRTVEGFRGTDGKVREVVTTDGTFETDMVV
LCIGFRPGTDLLKGQVDMLPGGAILVDDYMRTSHPDVLAAGDSCAVRYNPTGEAAYIPLATNAVRMGTLVGLNLLKPAVR
YQGTQGTSAIKLFDWNIASTGLTEAAAAGMNVEAVVIHENNRPEFMPTYDDVMLKLVFEAEGGRLVGAQVMSKADMTPAV
NTLSVCIQNRMTVQELAFVDFFFQPHYNKPWNLLNTAGLAAMQRLNETEKAVTEPERMEPGAERADARTPDKYGVA
>lcl|NZ_CP026520.1_prot_WP_042226883.1_3715 [locus_tag=PC41400_RS18810] [protein=hypothetical protein] [protein_id=WP_042226883.1] [location=complement(4174973..4175575)] [gbkey=CDS]
MMNKMIVSVLLGSLIVSGSLLLNGTAMRPVLPDSGMTAGKPVLPGDGGISGVKVPVPSGSEESRGEKVYVRLPVTTSANN
IEVTVHAVRLKAGTADFEVTIRNLSDRETITPDLTSGSVVTNEGLPGRSSEKVRAAKVNPDFEQAVIPPGKERYGWLSSD
AWSEEAPERLTLNVTIRGNNGEEQAFAIPVDCRGLTFRTL
>lcl|NZ_CP026520.1_prot_WP_042226886.1_3716 [locus_tag=PC41400_RS18815] [protein=alpha/beta hydrolase-fold protein] [protein_id=WP_042226886.1] [location=complement(4175603..4176946)] [gbkey=CDS]
MKSLSSLLKIEQFYSAFLNNRRDLFVYLPPGYSEESDRRYPVLYVHDGQNIFHPAFNGYSWNLHETADELVQNGLMEEII
IVGIANMGMQRSDEFTHELEGVDYLRDKVEIRPKGQLYEQFITDELMPYIDSLFRTKKGPEHTAMMGSSRGGQVTYHIGM
RRPDLFGKLAILSPYFYCVDPVTLEEARQYHTWTDKVPLSRIWIDLGRREGTLILEKHVREVTESLVGLGYKPDDELVYY
LDPTGTHSEKDWAARVASPLLHMFGNKGTPVKFSLEGEEIAGIRGPGMRLNPIAEFDSGFKMSLLRADYAAEDGTILEVL
EDGTLVPGREGKTAVEASFGGLSAARTIRVAEELKEHVALELFVHVPADTPENAALYSWVPLRKDASKPHAYTVRLDVPL
NAAFEYRISRGDGAVEADQAGQAVTRFYKAEADGRLELTVRSWKSPQ
>lcl|NZ_CP026520.1_prot_WP_042226887.1_3717 [locus_tag=PC41400_RS18820] [protein=alpha/beta hydrolase-fold protein] [protein_id=WP_042226887.1] [location=complement(4176943..4178301)] [gbkey=CDS]
MSSSRLIEINPFYSYTLKNERTIYVYLPPSYHSDDRKRYPVLYMQDGQHVFFKDRKGESWDVHLTVDELTAQGRMREIIV
VAVSHVEDARIAEYMHANPDGHNIFQTTNQGELYETFLVREVKPFIDATYRTLTEKEHTALMGSSAGGLVSYNIGFRQPE
TFGMIGALCPFFVSVDPATMEDRWLSAVYTDKKPLKIWMDVGDAEGFTVMEKHVRQVADTLMEAGYKPGGDFMYYFAVGS
GHSQKDWAARVHAPLLYFFGEIGTPVRIDLHGPERIGLQGPSCALNPVVHYDSGFMMTDLDADYEVTDPGIVEVTAEGRL
LPKREGNVTVRYKRQGLTASLDLTVVPHLSGTVSVTMFVEVPVCTPKTDKLYAGIELPMIGERLYGGTFQVPRGISFEFR
ISRGLGKHETDASGREIPYRKFTAREGLELSYRVEHWVDVPPDTEAKEAANG
>lcl|NZ_CP026520.1_prot_WP_042226889.1_3718 [locus_tag=PC41400_RS18825] [protein=LacI family DNA-binding transcriptional regulator] [protein_id=WP_042226889.1] [location=complement(4178328..4179242)] [gbkey=CDS]
MKKATIKDIAKAAGVSIATVSYILNDVKTQSISDETRVKVMKAAQQLKYVANRTAQSLKIKKTGLIGILLFQEQEGHYWS
DFKYAKTVGKIEQICGQLGYHVIFMQVRSQLSDYKMIMERNLDGVFLINADQEQFGTISNYFGFGIPVFVIDSYIDDSLF
HKIIPDFAQSFRACEAALGGRPEFLVTDAYNNEKLAGFIREQSGLEDERICVYDNPEQLNDFLSRFEGKPGIVINEFLAL
EVLKKTARIAAVCTCHCPQVLPGTVGRITFDLNDYQEIVHKMDRYIQDADYVDPEKFYYLSGTP
>lcl|NZ_CP026520.1_prot_WP_042226890.1_3719 [locus_tag=PC41400_RS18830] [protein=glycoside hydrolase family 13 protein] [protein_id=WP_042226890.1] [location=complement(4179275..4181113)] [gbkey=CDS]
MLLEAIDHRSGQAYCYAYDERTVHIRLRTKKDDVDAVHALHGDKYDWERTALETRLYKTGSDQLFDYWQGELVPPHYRLR
YAFKLTSGEEVVWLTEKGFGTEKSQDPLTFFDYPFLWPQEVLKPPAWVKEAVFYQIFPDRFANGDPANDPPGVRPWSDSD
PGSYDFYGGDLQGVYDRIDYLTSLGVNAIYFTPLFEAATNHKYDTADYKKVDPGFGDTALLKKLVAAFHERGIRVILDAV
FNHSGEGFAPFRDVREKGQESPYRDWFHIKEYPLRTEPRPSYHCFAFESHMPKLNTANPEVRNYLLDVAEYWVRECDIDG
WRLDVANEVDHEFWREFRKKLKALKPDFYILGEIWHDAMPWLQGDQFDAVMNYPATDAILDYFAKGEIDARELGNRINRL
LASYPQQVNEAAFNILGSHDTPRLLTRIPDEAKAKLAVMLQFTLPGTPCIYYGDEIGMQGGSDPDCRKPMQWDAERQNHG
LLAFYRMLIELRRTRRALSEGTFRFVYAEKDQLIVLRKNAEEELYICVNNSSREASCHVFEPESAYSDADAQFGTPPDLE
RGNRIRPEAEVILGQGSFVRTENGLSAALPPFGWLVASVPLWRFDTGQGDML
>lcl|NZ_CP026520.1_prot_WP_042226892.1_3720 [locus_tag=PC41400_RS18835] [protein=sugar ABC transporter permease] [protein_id=WP_042226892.1] [location=complement(4181151..4181984)] [gbkey=CDS]
MRKFKDNLTVTGIYLVFIALLVATVYPILWVIGTSLNPGQSLLVSRMIPDHPTFAHYAELIRQTDFVLWYRNTIKIALVN
AVVSTTLVAATAYAFSRFRFKGRQQGLMTLLILQMFPGFLSVMALFVLLLQTKLLDTHTGLILIYAGGSVSMGTWVMKGH
FDTIPKSLEEAVFIDGGSHADAFFKIIFPLSLPAVTFVALNSFITPFMDFILPQIVLRSSEKMTLAQGLYSMVATETNSS
FTLFAAGAVLVALPITCLYMYFQKYLIHGIAAGADKG
>lcl|NZ_CP026520.1_prot_WP_042226893.1_3721 [locus_tag=PC41400_RS18840] [protein=sugar ABC transporter permease] [protein_id=WP_042226893.1] [location=complement(4182003..4183301)] [gbkey=CDS]
MQVNPNNRRHRIRAAAGSAVLIGLGQLYNRQIVKGLLFLLSAQLFIWSYCPLALRGIAGLISLGDVPTKVVRGKVVPGDH
SIFLMINGVMTCVIVLLFIGFYVLNIADAWKNGELRDRGKPVPGFRASVKQAKFKHFPYLLLLPAFVFVLFVTVIPLVFN
VMIAFTNYSAPGHIPDRSLVDWTGLRTFYDLFAQKAWRSTFLGISLWNIVWAAASTITVFFSGLLLAMMINHPRIRFKKF
WRTAFMLPWAVPQFISILVFRVLLNGSFGPVNGFLASLGLPPVSWLSDPTMAKASILLVNLWFSTPFLMALMAGILTTMP
RDLYEAADVDGATAAQKFFGITLPLVKLATAPIFIMQFAFNFNNFNLIYLLTDGKPNNPDYYYAGSTDILISWIFKMTLN
QSQFNMASAVSIIMFVFIALFSLWNYRRMKAL
>lcl|NZ_CP026520.1_prot_WP_042226895.1_3722 [locus_tag=PC41400_RS18845] [protein=maltose ABC transporter substrate-binding protein] [protein_id=WP_042226895.1] [location=complement(4183338..4184624)] [gbkey=CDS]
MSKRLFSAGIVVLLSVTGLTGCGNDQPSPSAQPQEQGGGNSQQKELKPESGAKLVVWESKGPELDYLKTVAASFEKEYGV
SVKVEPVPAIDTVKKLTTDGPAGIGADVFSAPHDQLGNAVMTGLVLENDAFGEEDKKQLLPSALEGVSYSGALYGYPTAI
DTYALFYNKKAMPQAPKTYEEIVKFAETYNDPANKKFALMWDVGQLYQSYSFLAGYGGYVFGGKGTDPNDIGLNSPQAVE
GAKYLQSLKKILPLNIHDINDNIITGFFQEGKTASVINGSWLISSLSKANFEFGVAPLPLLPNGEHPVSFSGIRALYVNS
YTKYPDAAKLFASYATSKDHLIERFKMTAQLPPRQDVMADPVLTSDPYAPAFLEQAKHSVPMPSIPEMGNVWVPAGTAIS
SLWNENQDPAAVLTKAVEQIKTAMKTTK
>lcl|NZ_CP026520.1_prot_WP_042226896.1_3723 [locus_tag=PC41400_RS18850] [protein=RidA family protein] [protein_id=WP_042226896.1] [location=4185033..4185431] [gbkey=CDS]
MENSKIIRKNPAGVHSPVGNYTHLTKIPRNSELYVTSGQIGADQHGNIPDKMNEQIRNTFRNIQTVLHAEELSSEHIIKV
NIWATEKIDWDFLYEKWAELFGLDYPAMTVGYISELGLPEIKIEIELWAAKV
>lcl|NZ_CP026520.1_prot_WP_241688695.1_3724 [locus_tag=PC41400_RS18855] [protein=ABC transporter ATP-binding protein] [protein_id=WP_241688695.1] [location=complement(4185647..4187359)] [gbkey=CDS]
MLKEMFVHTKAHWGLLLTGALAVLFISFCEFIIPQLTKTVIDDIIPGKKADALLETGGLIVLTALVLGLLHFTSSYVMTV
VSQKAVLGLRNKLYTHTLRLDVKFFDRSRTGDLMARLTGDVNQLQELVSAESLSILADFVTFVAICGYLFYVDWQLALLT
VATLPFLFISSRFFSLRIKSAYRRVRQISAELNNFLQDTLSGIRMIKSFASEENEEKQFVKLSDRNRIAAVSASRLSALF
SPLTDWLNYIGMTLVLLFGAWQTMRGNLTVGEVVAYLAYLRLLQAPIRSFSRLVTKIQQSAAAFERIGDILDTVPEVRDQ
EGAIVLPSVKGEIVFYDVDFAYDEGQPVLESFFLKLRGNETTALVGSSGSGKSTIADLIARMYDVQRGDIYIDGYPLKAV
TQQSLRSQLGIVSQEVILLNGTIRENIAYGQPAASGEQIEAAARAANAHEFITALPQGYDTPIGERGVKLSGGQKQRLSI
ARAFLKDPRLIILDEATAALDTESEQKIQQALSRLLPGRTCLVIAHRLSTIQNADQIVVLEHGEIVEYGKHDVLLRRNGR
YKELYRIQFP
>lcl|NZ_CP026520.1_prot_WP_042226900.1_3725 [locus_tag=PC41400_RS18860] [protein=response regulator] [protein_id=WP_042226900.1] [location=complement(4187762..4188790)] [gbkey=CDS]
MYTILLVDDDSTTIKLMRYMLENSRFPVLRIFEASDGKEALRIMREYGADLVITDISMPYMTGLELCRQLSGEFRYTPAI
IVSEHDKFEYVREALVCGVKDYLLKPLSKDRFLSSVRRTLEELPETVHTIGETSDRWRGLAEAWVEAVLQDNRSAQEKAG
RKWLASLHGVPPYLCLKWHSAFHRILLQSAGAKLEREVECSLPPYHGQNGTDLTNWITADFKALQHAVKLAAYTDNQRTI
YNAQKYIQQHYNKEITLEELASSVGYSAPYFSYLFKLKTGISYMAYRTQLRLKRAQELLLKSDKSMPQIASEVGYNDVTS
LIRAFKKELGYTPSDYRKSGLV
>lcl|NZ_CP026520.1_prot_WP_042226901.1_3726 [locus_tag=PC41400_RS18865] [protein=extracellular solute-binding protein] [protein_id=WP_042226901.1] [location=complement(4188975..4190300)] [gbkey=CDS]
MKRMLLLITTLLLLTTTAVCGGWGRTQAPDSGGTSRGENKNGKKTVIVSMLFLDEVFREAKKKFEQKHPDVEIRLTYVQT
KDNWKVGDQERFIKSTNAEMLSGKGPDVLVLDPLPMGKYASSNALVNLSRFMDQDPSFHKDLYFGNILDNLHINGVLYGM
PMRFMLETMVGDEGAVRKAGVGIDDTVWTWNEFAETAKAMKKSGSPAAYAASPPSMLKAMVKADYSSFVDEPNRKAKFDQ
ASFIRLLNEVKALFDGKVIADKPAVVSFKSAYIQNPEEFFSFPRALYKQPKLYRSPLAVGQKPGGFYMPEQILGINENST
VKTEAWAFIKFMLSDDIQGVNGLGGFPLNKDVYGRQLDQVLKNGKVKIEDGPEEGGSVDVNPADIEFQKELISGLIHPIP
TNTSMIEDVLLKESKAFFTGQKSAEEVAKEIQDRANTYLNE
>lcl|NZ_CP026520.1_prot_WP_156972785.1_3727 [locus_tag=PC41400_RS30105] [protein=hypothetical protein] [protein_id=WP_156972785.1] [location=complement(4190610..4191263)] [gbkey=CDS]
MVKRELSMLFFIFRTQTILICLPFLVYTFACLGNYYGYFNGLEFMVFVGFWIHVFPAVAVTYQIGSLRQQELFISLPLSS
VRYGLIYPCFLSVLYGSFYETALLICINSNNGFAKTADMAVNSFPSFLLVFFLTVTLVSLFKNSAIGLCLTLFYVFFGLF
TGGAGQGPFYLMQWYRPKINTDVNDFVIVQYAAVIVLCVLHILFVKYRHRFYLLKIT
>lcl|NZ_CP026520.1_prot_WP_042226904.1_3728 [locus_tag=PC41400_RS18870] [protein=ABC transporter ATP-binding protein] [protein_id=WP_042226904.1] [location=complement(4191245..4192114)] [gbkey=CDS]
MLQIKGLKKSFGKREVLQEINVAIGTGVTGVLGPNGAGKTTLFRILATIYEPTQGEIKLDEWTWKHDLEKIRELCGYLPQ
HVGLFPSITVYEYLEYVGQLRKLTDSKKRRQRINQVLQEVNLESKRDDKIKKLSGGMKQRLGIAQAILHEPKLLLIDEPT
AGLDPEERLRFRSLLRQQAQHRVILLSTHITEDVAMTCDRVCMMKNGHLDFYNALANVTQMAENKVWSMELSQDEFTKIR
DSDECRIVATSETGGRLSVRVLANQRPIEHATPADPSLEEGYMIWLSEN
>lcl|NZ_CP026520.1_prot_WP_042226905.1_3729 [locus_tag=PC41400_RS18875] [protein=ABC transporter permease] [protein_id=WP_042226905.1] [location=complement(4192145..4193716)] [gbkey=CDS]
MLKEFKQLMRQPQYLLLFIFAELASVLLPMKYLEGLRDFPSLIFFVMGFNLLFLLYGAEQARSDHNNHIPELIASLPKGK
WYYLQKILYWFIFAFIWYVIFLVCALLYVTYGIHDDATLFQFRITLVYTFFNWLVPFFVSLVLGYSIYTLWPSLAAYLVL
ISIWILLGPYNSFVGFIPPSWLPLLTVGDTNLSITQSYYQSEHMILNDGAFWQRTLAVLVCFFVFFVSIPLLRKGRKARI
LMIVCLLVFTCLAGFAPSQKITDVTSEYKFNDAVPDYADFTISSYDMQINHGREDHLFSYVADLNVSTKSDQIPLALWDF
MDVQSLSWNGASVPFSRDRNLLFVQLPSGHTETGILSFHVRSEKYGDMNPTSFELLSTLPWYPMDPKEAKDPLHHAIKEK
FSVKVNVKDFRGLVTNLSKNENQTLTGVTYGPTLLYGPFTQETDLTIPKFISLKEARAAVSQVAEIINSGRQPSDKVVSL
PQHGFIVNSRSIFSANPDEYFLTLTKSKIDSNIFRVFFPEKEK
>lcl|NZ_CP026520.1_prot_WP_042226907.1_3730 [locus_tag=PC41400_RS18880] [protein=hypothetical protein] [protein_id=WP_042226907.1] [location=complement(4193805..4194179)] [gbkey=CDS]
MLKSKIVALGVSAVVTLSFAVTAFAAPNHAYTVKGLNTSSATSGYVGGDGIYGRYDAALRQGDNGRVTLYKQRTLLPDQA
VDFIDLGTSKTYDLKTYYMNEGTLYYIKATGDNNDEVAGNVWNY
>lcl|NZ_CP026520.1_prot_WP_241688696.1_3731 [locus_tag=PC41400_RS18885] [protein=hypothetical protein] [protein_id=WP_241688696.1] [location=complement(4194837..4195991)] [gbkey=CDS]
MMTRRSMLSLPLLLAALAMPAGCAGLPKAKTIVIGDTAEDAHTEEESTDFETNKIYRLSLPGNKAVDVWGWMSSRLLLGS
YKNQVTKQDIQLVDYRNLTAVSLTPLAGEAQTGKPSPDGKYMAYTGQSRTGETLYVLDLNTSQSFAPGVPDKGKGITSAM
TWSNNGRYLSFTVKSREYGTESIVVYDMNSRTSQEIPLPRWQENSEAPAPRALKRDSKYTVISVKFADDGSTALMGRIGE
DGSYVTLGAVTEDKWTRKFEHSASERQFDYVNDSQIIFLDAKNTLTLYDERDENTVALSAGVSTFRLSGDNKTIAYAKDE
ESLYVAKLQGHNMIKEKLVYKGLVLSQMDWSPDNKKILITGWKSYDRAPATGNPEQTPLVIEFK
>lcl|NZ_CP026520.1_prot_WP_042226909.1_3732 [locus_tag=PC41400_RS18890] [protein=ATP-binding protein] [protein_id=WP_042226909.1] [location=complement(4195988..4197451)] [gbkey=CDS]
MFWTIRAKFLIGLFVIFSVSFLLLHHFVVRIIETSNEKIITQDLIQLKNNSSIYVKQAFLINHFNTDDVYFQQMAEEMVR
DLNHVTSSHTSAYSVAGDLIFASDEAKFLSANPDDLQYALVGKTAYRIRYTDQKAEVFYAYPVIVEGKKVGILRFGKDFS
LLYEQRKHILDVFSSVTVAIFIAAFVFSYLLSRNITIPLIKLTKASSEVTNGNLNIKIGFRRKDEIGQLAHNFGNMIVQI
KKQIQRIKKDRDRLEELNRHRKQFFDHVTHELKTPLTSIIGYAELIREVGSRDEAFFAKGMNHIVSESKRLHEMVLQLLT
LSRETGGQDATETIEPGLILRDVCDGMLLKAQKYNKFIHCEADENLSLVADPAKIRQLYINIIDNAIKYGRSPSTVLVKA
EKLPGFVRISVKNEGETIPSGELAKIFEPFYRVSGPDDREAGSSGLGLSISKAIVDGLGGTIGIKSAEQVTTVIIELPYE
EAGEGEA
>lcl|NZ_CP026520.1_prot_WP_042226911.1_3733 [locus_tag=PC41400_RS18895] [protein=response regulator transcription factor] [protein_id=WP_042226911.1] [location=complement(4197456..4198163)] [gbkey=CDS]
MENTREKTKLLIIEDEEAIADLLAYGLNREGFEAFTAHSAAEGLLKIGQVRPDLLLLDWMLPDLSGLEVCRQVTATYKIP
IIMITAKSDITDKIQGLEYGADDYITKPFDLREVAVRIRTVLRRVSQANVMPEAPAAAVIRFRDVEIVEQERTVRIGGED
VDLTPKEFDLLLTLCKYRGKIFTRTELLDFVWGYDYTGDTRTVDIHIQRLRKKLGAGDLIVTVFGVGYKFEKQVD
>lcl|NZ_CP026520.1_prot_WP_042226913.1_3734 [locus_tag=PC41400_RS18900] [protein=MFS transporter] [protein_id=WP_042226913.1] [location=complement(4198361..4199581)] [gbkey=CDS]
MNNQAPLTNASVVRRNYSLMTVILCWSGLAVMSSLFVTIPLISVFARLYDITPTEAAAAGSTFSIGFALGCLIYGALSDK
YGRKKVIFTGLLVLTFISFLLALVHSFAWIVVLRGLQGAAAATFSPVALAYAVEMYPAEKRVTTIGFISTGFLTAGIVGQ
VVSSLISEQYGWNAVFYVLAGVYALTAVLVYVFLPKGGIQQAHASIWEPVRQIGQVFVRKQLVLSYVVAFVLLMSFVNMY
TVLGSYLSGAPFGLDKQGVLVVRSIGIAAMLLSPFAGRVSKRFGIHMTLRGALALAAAGLASMGLISSLPVLVAVSLVFV
AGIALAVPSLVSLVGQLGGKLRGIAVSVYTFVLFAGTSFGPVLSLSFMKSFSYPVTFLLLASILCIGLAAACLIDREEPL
EVKKSR
>lcl|NZ_CP026520.1_prot_WP_042226915.1_3735 [locus_tag=PC41400_RS18905] [protein=MarR family transcriptional regulator] [protein_id=WP_042226915.1] [location=complement(4199652..4200212)] [gbkey=CDS]
MNVNRKPAKEYTTEQQLPKLGTKPYRDLLERTANSDTDRDSALTGLLMLWLGDSVLDVMDIHLSDYDITESKMDILLLIS
LHQDKELVTPSSIADRLGIRRSSVTSLLNWLDKRDLIIREPYSKDGRMTHIRLSPEGSGLVKRVLPAFWSTCASLVDELD
KEERELFNKMLVKLNRGIEKRLGTGR
>lcl|NZ_CP026520.1_prot_WP_053228639.1_3736 [locus_tag=PC41400_RS18910] [protein=S-layer homology domain-containing protein] [protein_id=WP_053228639.1] [location=complement(4200534..4203896)] [gbkey=CDS]
MSKKFYHVLTVLTVFVVLFSSTAGSLFAAAAPQQAASPAEQIAEQRGSGDVQPDQGVFTATYGVPKDRSTLVQWNFRNAG
AGDNGIYRPAGGVYGTASTFRNSGGVFEYVEDDTLSYQGWDEGNGVKHWLAAVPTKGFRSITLSSQQNSSGSGPRNFKVE
ISQDKRTWTPIADLVMAKSSYNCSGDSCKLNELPLPSEADNQPQLYIRWIVSSDEKTSGGPGIGGSGSSRIRAVEVKGER
ISGSDISEPTVAVAEFPGDKSEEITADRELSLKFNKKIALENGKQATVADENGNVSETFLQVSGDTLTIKHTPFAYEKTY
TVLVPKELVKGAEDGYPLAGDITWSFKTRTSPKTPKLLNMTFNGDPKTSMSFAWYTPPDIPGTVVQVIEASEMKDGVFPE
AKASAYSGSSEVIESYFVKSDRDTGNKTKFASHKVTADRLKPGTAYKYRAGNGEPDGWGPVGSFTTDGAADEDFRFIVGS
DSQGSSKSSFELWQDTFRKAVDKAGQPKFFMVTGDLVDNGDLEQQWQWLLDTAKNELAHTPYVPILGGHEVHDYEGDENT
ENNNFYYHFNLPKQIIPGTHEGSIYSFEYGDALFIQFNTQFEGGLDPSGQVDWVDPEFTAQLEWMKNVVARTDKKWKFVS
LHKGPYSAGDNAGEWEGERIAFYKKHLIPALDQMGIDVVFEAHDHMYMRSYQMLNDVPMKDIKDDGAGNALNPPGTVYLM
TNTVGSKFYEKYKGYDDYFAKINEQPFKKMFTDVSVSKDVMKFTSYTAVKGEDVKPYDEYGIKRTNEKPAKVENAAVQSG
SGGTVLSWNGPKNSKEPVKGYRIYEKDDKIRPYWSEYVPDETKSGTYKLELGKLDPKQSYTFVIKAAGVRTNSDAVQIGI
PKDNGSSGGGSSGGDGGSSLPPVTTVPPVTGTKPPVNPDGKPKPEPEPEPKPKPEPARKTFTDVDSRYAWAKDAIELLAG
KGVIQGVTADTFEPARSVTRGEFVTLLVRALGLKADTAGRFGDVLPGAYYYEAVGTAKALGIAEGTGPDTFDPGALLTRQ
DMMALIVRALNAAGYGIQSGSTRDLAGFDDASELAPYAVESAAALVRERLVEGSGTGIMPYGLTTRAEAAVILYRVTSFK
>lcl|NZ_CP026520.1_prot_WP_042226916.1_3737 [locus_tag=PC41400_RS18915] [protein=glycosyl hydrolase family 8] [protein_id=WP_042226916.1] [location=complement(4203993..4205348)] [gbkey=CDS]
MKKKLLELSTCACLLGTLIAGPGTASAQETVPASSIQTEPVYGESSNVSIAASALKPFPQHVSYTSGTIKPNHFTQDQMD
QQVKKKYDEWKSRYLVKHPKISNQYYVFYNREKIVDDPKNVVSCSEGHGYGMLITAYMAGHDASAQTYFDGLYRFYKAHP
SSADAALMAWQQAKDSSGNIVDSSGSYSATDGDLDIAYSLLLADKQWGSSGEVNYKSEAVKMINAILKRDVHAKQFHLKL
GDWASDSDSKYGPGTRPSDFMLHQLKAFKDATGNADWDKVSDTTYKIINSVYKKHSPKTGLLPDFVVRKSDGTYAPAPDK
YLEEPTDDEYSWNSCRTPWRIATDYLISGDTRALEQLRQMNGWIQSATGGKPAKIGSGYDLKGNVKKADHAIAFTAPFAV
SAMIDSSNQQWLNKLWSDIVDSPTSDSYYFDNSIRLLSMIVVSGNWWTPGE
>lcl|NZ_CP026520.1_prot_PC41400_RS18920_3738 [locus_tag=PC41400_RS18920] [protein=NAD(P)H-dependent oxidoreductase] [pseudo=true] [location=complement(4205905..4206436)] [gbkey=CDS]
MARAAENAAKEAGAQVRLFRAAELAPEAVGASSAAWRAHLDETADVPEASLDDIAWADSSARRPVSAAWPLN*SSFSTPP
APCGHEGS*RTRLYPP*RAQATCTADRKRRSCPCI*ACTTGVYGCVRYGRGRESVRHKPDGGHERQFRRQGTGCR*PSGK
TGRYGCGNAAERAGSTV
>lcl|NZ_CP026520.1_prot_WP_042226918.1_3739 [locus_tag=PC41400_RS18925] [protein=ABC transporter substrate-binding protein] [protein_id=WP_042226918.1] [location=4206574..4208193] [gbkey=CDS]
MTDISSLHREPVSAPHLFHIPLRITEVNSQVKDLGRSERTGCSRIFVVLGGSGVLHIQEEPYEVSRGSLYLCDDFQKLDV
TSHGLLKGVLIEYRRLGTADPDTGALQCPEPLNGCPDKLLRQADELQKLWREPKLGSPFRIQQLFIELMNGLYEELASRE
RTGRSWLEQSVSFMQAHFVEDLTRDQLAERANVSPEHFSRAFRKYTGRTFNEYLTLLRIRGAQKRILTGSPDLNTLAQEI
GYKEGAYLSRRFKESVGVSPTVYRGKSKRIVTLNQNYTAILTVLGITPELGVYTGWTEKIAGGPEIPERQKFNPYGCTSA
SFYETVAAARPDIIISYDTQEENNSLLPLAPVLDLPFKTMSWREQFRMIAGIVNKRREAEEWLALYDKRLIPINRQLDRQ
LGSRGTAAVWEIGPDKAYGFGSSHGRSCQILYDDIGFRPPERMLEQGLSWRGYVETSMEAIVQYAADHIFITALPCDSHG
RQRLMRLFRSPEWLALDAVRRRRVYLLDQPELFFGFDPLSSLEQLRVLMNVLAPDHKFT
>lcl|NZ_CP026520.1_prot_WP_042226920.1_3740 [locus_tag=PC41400_RS18930] [protein=ABC transporter substrate-binding protein] [protein_id=WP_042226920.1] [location=4208283..4209293] [gbkey=CDS]
MFSHQRFALYRLQAAATAVLLLAAVILSGCGGREDSAAGSTVPGTPNVQASDVSGTGTTKETAFKKISTVKGEIQIPVRP
QRIVAEEYLGSLIALNVVPVGAPGLTLNNYYYKKALNGVADTGTYGKLSVERITGLEPDLIITGNAETYEQLSKIAPTLV
IPYGTLKNTREELEYFGTLLGKETEAKSWLEDYDRRIAAAKSKVDAVLPADASFTIMESTDKSIWVYGDNFGRGGQPVYQ
ALGRTPPAAISDELMKKQWAEISAEVLGRYAGDYIIMTANNKTAEDFRTDPVWSTLPAVKNGKLYVWPEERSWYYDPLAV
LSQTEELAEWLVKTGK
>lcl|NZ_CP026520.1_prot_WP_042226922.1_3741 [locus_tag=PC41400_RS18935] [protein=MbtH family protein] [protein_id=WP_042226922.1] [location=complement(4209703..4209900)] [gbkey=CDS]
MANPFDDQSGSFRVLVNVEGQYSLWPAFIRTPDGWTAVLEEGTHQECLEYIESSWTDMRPASLRE
>lcl|NZ_CP026520.1_prot_WP_042226924.1_3742 [locus_tag=PC41400_RS18940] [protein=non-ribosomal peptide synthetase] [protein_id=WP_042226924.1] [location=complement(4209893..4217143)] [gbkey=CDS]
MIGSALRLPLSGAQQGIWYAHQMDGQRSVYNTAEWLEIHGDINEELLEQALRLTVREAEWLHARFGEDGEGPWQELGPET
DWALLKIDVSGMKYPLAEAEAWMRQDLQKSVELAEGPLFTQALIRLASDRWCWYQRIHHILIDAYGYALLHRRASQIYTA
LAAGGPVPPHNFGSPEQVLREEEAYRASGQHERDRAYWLDKLGDAPDSSVMGRGGNSPDGSPLRQSGRLAAAASRVLTES
AAEAGCSWAEAVLAASAAYVHRMTGSADIRLGLPLMNRLGSSALNVPGMVMNVLPLCLTVRPETSWGDLRQQVAAELRKL
RRHQRYRQHDLVRDLGLLGERRRLYGPMINIIPFVETPSFASAGTVRHSVYAGPAEDLSLQIYEASDGDGLQIDFDANPG
IYSEQELSGHRRRFLSFLESAAFANPDTPAGLLDTLLPSEKDLLLNRWIDTACDVPGDSPAELIRRRAELSPHASAVVCG
GRTLTYAELDRQAAVLGAELRRLGIGPERLVAIALPRGEAAVVSILAVLRLGAAYMPVDPDYPAERIVYMLEDAQPACLL
TDTESMRALPETGKISLLLLDQLDKVDRAESPGSIPETGDHADAECLPLRPSPLNPAYVIYTSGSTGRPKGVPVTYGGLA
NLLDAMQRKFRLQEGDSFLSVTTIAFDISVMEILLPLIGGACLDIASRELILEPGALARRIRSTGASIMQATPTLWQSLV
SARPGRLDGLRVIAGGEALPLSLQLALQELGCEVHNQYGPTETAIYSTAAALPGGSMIKPPIGPPIANTRTYILDAGLQP
VPPGAEGELYIAGAGVARGYLNRPVQTAERFVADPWGEPGSRMYRTGDLARWLPDGSIDYLGRADHQVKLRGFRIELGEI
ESRIAGQPGIAQAVVVVREDRPGERRLTAYVVPEELHEPDHAQLKKQLQQELPEYMVPLVWVTLASLPLTPNKKVDRKAL
PAPQMSGTAKEASARTPHEELLCQVFAEVLGLSSVGIDDDFFRLGGHSLLAGRAVLRIRETTGVDVRLSGIFEAPTVREL
AQLLGRKGIAIPPVVPAAPGAVVPLSRAQRRLWFLYRLEGASSTYNIPLVCTLSGKLDRQALKQALLDVTGRHEPLRTVF
PETPDGMPQIRIFDREAAADRFVFQVIPAAGLEEDNWQEALDREVRHGFELSSEIPLKAVLFEREEGEYVLLLVVHHIAA
DGWSLSVITSDLSSAYAARIAGRDPSLSEPPVRYGDFTLWQEKLLEEESEPQSLMGSQLAYWKEALRGLPEELPLPADHP
RPSVSGQAGGVIPFRIGEELHAQLLDLAAEGKASLFMVLHAALAALLARLGGGEDIPVGSPAAGRPDEATGSLVGMFINT
LVLRADLSGQPTFRELLARVRQTDLSAYENQELPFERLVEALSPARSRSRHPLFQIMLVLQNTPATRLDLPGIASKLELR
GTGSSKFDLTFELTEHRRPDGEPCGIEGLLEYSKDLYLPHTAEALTRRFLMLLEQAAAQPDQPVGRISVLLEEERLALTA
KPRPEAAKQTIAGGKLSAPGRTLTELFEGQAALRPDAPAVVYGTQTLSYRQLNERANRIARLLMEEGAGPEQFVALALPR
SADLPAAVLGVLKTGAAYLPLDPDYPADRLAYMLEDARPSILVTSADAAASMPERSGLKIVVLGDFACEERLRKLPESDP
EPDLRAGRHHALHPAYLIYTSGSTGRPKGVVIPHENVIRLMESTEEWFGFDETDTWTLFHSYAFDFSVWELWGALLYGGK
LVVVPFSVSRSPSEFLGLLEEQRVTVLNQTPSAFYQLIQADREKSGQASDLSLKHVIFGGEALDPGKLEEWYSRHPDNFP
RLVNMYGITETTVHVSYYPLDRRSAEAGSGSVIGVPIPDLDVYVLDDNLEPVPFGVIGEMYVSGAGLARGYWRRPELSAQ
RFVADPFGPPGSRMYRTGDLAKRFADGSLEYLGRADQQVKLRGFRIEPGEIEAVLEKHPAVAQAAVVLREDQPGDKRLAA
YIVPAQSGEVPDAGMIRLHASAALPDHMVPSGIVVLDSLPLTVNGKLDRKALPAPVYASANGSRSPRTPQEEVLCGLFAE
TLGLEDVGIDDGFFELGGHSMLAVRLMSRIREVWGKDPGIGLLFETPTVAGLAARLEEDTGDGALEMLLPLRSNGSGAPL
FCVHPAGGLSWCYAGLMKHLNPDVPIYGLQARGIAGPDRLPQTLGEMTEDYIRCIRTVQPAGPYRLLGWSLGGNVAHAMA
VQLQKEGESVEFLAMLDAYPSHYLPLRGEPDEEEALTALLALGGYDPDSIGDEPLTMETAIEILRGDNSALASLSVETIR
QLRVTYENSVKILAAYVPEKYEGDLLFFHSTIIPDWFDPIDPDMWIPYVGGEIERHDIECRHKDLCQPGPLSIIGRMLAG
KLSSAKAKEEEGSFYG
>lcl|NZ_CP026520.1_prot_WP_042226925.1_3743 [locus_tag=PC41400_RS18945] [protein=isochorismatase family protein] [protein_id=WP_042226925.1] [location=complement(4217143..4218195)] [gbkey=CDS]
MAIPSVQSYPMPTAEDLPANRVSWTPDPARAAFLIHDMQQYFMDCFEPGQSPAAELTEHIRLLANRCRELGVPVIYSAQP
GAQTPEQRRLLQDFWGSGIGPDPREQQIIAELEPGDNDIRLTKWRYSAFQKTDLADQLKAMGRDQLIIVGIYAHIGCLMT
ACEAFMRDIEAFMVSDALADFSREKHDLALKYAAERCAVTLTTEQLLAQWAKETSETVNLGQAEAGAVSNAAAGIAETAA
ADGRSDGPEAAGAPDPYAFALSALRQQVAGLLRERPDAIGDEDNLPEYWGLDSIRIMTLAERWRSLGASVTFAELAEEPT
LLAWSRLLASRTVTFVPNGDYYDAAARGER
>lcl|NZ_CP026520.1_prot_WP_042226926.1_3744 [locus_tag=PC41400_RS18950] [protein=(2,3-dihydroxybenzoyl)adenylate synthase] [protein_id=WP_042226926.1] [location=complement(4218267..4219979)] [gbkey=CDS]
MSEDYPYWPDEFVRKYREAGCWRGETFGGMLRERAAAHGDRSAIVGGEERVTYAELDLNADRLAAGLKGLGIRKGDRVVV
QLPNVPVFFDVIFALFRLGALPVFALPLHRRSEIVHIASYSEAEAYITIDKYAGFDYRTLASEVKAEVPGLRHVLVAGNP
GDHLSLEETYAHEIPAIRGKLHPLQVDRSEGQASDVREEQPGTPDAGPAPGDTAFLQLSGGSTGLPKLIPRTHDDYIYSL
RRSAEVCGLDGGSVYLAALPVAHNFPLSSPGALGTLYAGGTVVLARSGSPDEVFPLIRKEKCTIAAVVPPLALVWLEAAK
TRGETLHPLQVLQVGGAKFSEEAAKRVKPELGCTLQQVFGMAEGLVNYTRLDDAESVITGTQGRPMSPLDEVLILDDEDR
PAAPGTIGHLLTRGPYTIRGYYKAPEHNARSFTPDGFYRTGDLARMTPEGNLVVEGRAKDQINRGGDKVGTEEVENHLLA
HSAVRDAAIVSMPDEFLGERSCAFILSRDPHTGGDELRSFLRRRGLAAYKIPDRFEFIDSFPQTGVGKVSKKELRQRIAA
LLLEEAGVKS
>lcl|NZ_CP026520.1_prot_WP_042226928.1_3745 [gene=dhbC] [locus_tag=PC41400_RS18955] [protein=isochorismate synthase DhbC] [protein_id=WP_042226928.1] [location=complement(4219976..4221286)] [gbkey=CDS]
MMLGQKEKLLSPADLLDKYSEGSAFYFGSPRRTLLAEGELVRLSVEGKGEELERLPRRVADLLAGVRAGNAAEPVIVGAI
PFTVNSGVRPELFVPRHTEWAGPLADAEAVQADVAGDTARSVFDGEVDGGTFKIVSFPTESDYEAAVRKGLEQLRSGKLR
KVVLSRSLRLENCPPVQTGRLLRRLAASNTRGYTFAVPLRASGSGHNPGQGQGTAPASAAHTLIGASPELLVARSGQLVT
ANPLAGSAARSSDPAEDKRRADALLLSAKDRHEHAVVVDAVASALRPYCRKLDVPAEPSVIGTKTMWHLSTKVTGELLEP
AASSLELAVAMHPTPAVCGTPTGPARTAIRELESFDRRYFTGMVGWCDAAGDGEWAVTIRCAEIAEDAVTLYAGAGIVEG
SDPAAELAETSSKLKTLLLAMGMDPDTGSGGGGKSV
>lcl|NZ_CP026520.1_prot_WP_206735020.1_3746 [locus_tag=PC41400_RS18960] [protein=2,3-dihydro-2,3-dihydroxybenzoate dehydrogenase] [protein_id=WP_206735020.1] [location=complement(4221310..4222095)] [gbkey=CDS]
MPFTGISGKVAVVTGAAQGIGEAAAAVLGSLGAVIAAVDRQGDKAAQAAEALASSGVSAAAYTADVADSRAVEEVIGRIE
REQGPVDILVHAAGVLQYGTGVTLGDEEWERTFAVNTTGAFYVCRSVISRMSKRRRGSIVLIGSNAGTVPRYGMSAYAAS
KAAAAMMLKCLGLEHAAEGIRCNIVSPGSTDTPMQRGMWEDESGEFKTISGSPELYKLGIPLGKLATPEDIAWAAAFLSS
DQAGHITLHDIRIDGGATLGR
>lcl|NZ_CP026520.1_prot_WP_042226931.1_3747 [locus_tag=PC41400_RS18965] [protein=alpha/beta hydrolase-fold protein] [protein_id=WP_042226931.1] [location=complement(4222176..4223042)] [gbkey=CDS]
MVSRVGIPRRTFQGETANGGVGIPNAEQRVMRSSRTNLSYRIMTAIPPGPPPSTGYPVFYVLDGNSVFATVTEAVRVQCR
RPEVTGVKAALIVGIGYPTDEPFHPQRYFDYTPEKTAKYTRKPDGTMLPEQGGAAAFLDFIQEELKPVIAREYPVDLTNQ
TIMGHSLGGLFVLHTLFTRPEYFRTYIAGSPSIHWNREWLDEREKLFTAALDAGSLQKSPGIRALIAMGELENAHPSGNY
RMAREMAVRLSKLENRGLSAQWHSFEEEGHVSVLPGLISRGLRFAFRP
>lcl|NZ_CP026520.1_prot_WP_042226932.1_3748 [locus_tag=PC41400_RS18970] [protein=ABC transporter substrate-binding protein] [protein_id=WP_042226932.1] [location=complement(4223241..4224242)] [gbkey=CDS]
MRNSFGILNKIVTVKVLAVICTALMLTACGANPNAGSADGKAGDTSASAAPTGAAGGQERTFTDEFGHTVKVPASPKRVF
APYLEDSLTKLGVTPVAQWSAGGQGNSYLKEKLSQVPKLDFSSGAPSPEVVTELQPDLIILQTESYASNGVYEKYSKIAP
TYVFKNASSDPAASLSKLGELLGKTEEAKQALQAHDKKVQEAKAKLQQAAPGKKAAVIRFASRGVSLMGGKYLAGAVIHD
KLGFGMSKLVDGKNSAAISLEVLPDLDVDYIFMMNQYGQGTERMKEITDSPFWKKIPAVQAGHAYEVNSEYWLGSGMIAD
EMMIDDVVKLIAK
>lcl|NZ_CP026520.1_prot_WP_042226934.1_3749 [locus_tag=PC41400_RS18975] [protein=AraC family transcriptional regulator] [protein_id=WP_042226934.1] [location=complement(4224287..4225513)] [gbkey=CDS]
MKPGKSSARIEGRGERAAATRPYLTDIRVYMDSHYHEQLTVRQLAERVHVSPKYFVDLFKKTFGQSAMDYLTDLRINRAK
RYLSEPEPLLLREIAQRVGYKDEYYFSRKFKKEVGMSPTEFARSSKHRIAAWSPDMTGYLIALGLTPLAAPLDPKWTAHY
YYSYRKKIRYALKLTDPYVTGTFEENADLMARLRPDAVVAGDGLSDEQQDTIGKIAPALFVPTSQELGWREYLRLIAVFL
KRTDEAEQWIRGYDKLADEARIQSEAALGDDGILIVRLYGKGLYSYSNRGMEDVLFKDLNLRQVSSGKQSAPIRPEDIRA
LDPSRILVLVCPESESRAYWLALQHSPEWKTIEAVRSGNIYPISADPWLEYSPVAVARMLDEAMLLFTGKCPNAFLDHVH
GGSLPADL
>lcl|NZ_CP026520.1_prot_WP_042226935.1_3750 [locus_tag=PC41400_RS18980] [protein=WG repeat-containing protein] [protein_id=WP_042226935.1] [location=complement(4225633..4228137)] [gbkey=CDS]
MRADEMQVTSLVSGFLPQGAELMTIEKPAAQPAIHSADLTGDGIPEIAAVYRLNGEMYLLVLTCRDGVWQKAATVKGTGY
GVTLLSSAPVLRAGLNNLLVGWQIGSIWSKLSVYEWTPEGLKDAAPADMSYSYIEAEDMPGLTGRNGKAEVALWIHDTGE
AYRVQVLRWSGGAFVPAPDVYPYYFPRVVRYYEAMTRQHPDYTFYWYYLADAQYRAGLPEAALVSVRKALSFPNPYPSRE
KLLELRLNILNGLGRSPQARMPALYPASLKTTSGLKWGYIDPEGRLAIRPVYEDAADFQQNGLAVVGRDGKYGLIDRWAH
FVVRPVYNSINPFSERRAVVIDSEGFKLIDEKGKVLTRRAYPFIANMENGRAVFYVTEGGSGDTGTSRYGYLDAQGREVI
PAQFEEANDFSDDRAVVKIRDSDYALIAPDGRKLASYPYAFVGPHGDGLLAFQQTSGGKYGYINERGKVVIAPSFTVGLP
FHDGRAVVSTSEDYKWTYGVIDKAGRYVIKPAYNDIRDLGEKRFALGRAIDPEQPFLGSVYAIADWDGALLSDFLYDEVS
DFKDGLASVTEAEQTYFIDRSGKPAPGYPRVEGSGTLTLERGGLIKAYTDRRLAYLNRAGKVIWRQNTVIPLRPPYLVRE
GKYKPNRDYLVYYPQVEGLKDETVQRSVNDRLKEWSQVKPVPGDVKLDYSYSGDFQVAFFKDKLLELELTGYKYPFGAAH
GMPSKTYAILNLENGRMYALKDLFKPGSDYVRILSAIVGRQIKEDPQYAYVFPDSYKGIKPDQPFFVTENALHLYFFPYE
IGPYAAGFPTFTIPFARIGAIIDREGQFWKAFHA
>lcl|NZ_CP026520.1_prot_WP_042226936.1_3751 [locus_tag=PC41400_RS18985] [protein=glycerophosphodiester phosphodiesterase family protein] [protein_id=WP_042226936.1] [location=4228361..4229344] [gbkey=CDS]
MKKIKRILKSKFFLFAALLLVFFYVNNSSLFVKREAGAPYLLAHRGLAQTFTMENIANDTCTAERIYPPEHPYLENTIPS
MEAAFQAGADSVELDIKPTKDGQFAVFHDWTLDCRTPAAGKTSDYTMAQLKELDIGYGYTPDHGKTYPFRGKGTGLMPSL
GEVLKLFPDRSFLIHIKSNDAAEGKQLADFLSVRPKEQVERLTVYGGDEPIDALKAELPGIRVMSLATMKSCLIPYIAAG
WTGYIPSACERTQLHIPEKIAPWLWGWPDKFLNRMDEANTRVIVVGGDGSDFSSGFDSPEDVKRLPSGYSGGIWTNRIDR
IAPLYGK
>lcl|NZ_CP026520.1_prot_WP_042226938.1_3752 [gene=dhaK] [locus_tag=PC41400_RS18990] [protein=dihydroxyacetone kinase subunit DhaK] [protein_id=WP_042226938.1] [location=complement(4229462..4231225)] [gbkey=CDS]
MKKIMNKPENLVKEMCSGFVLAHPNLEFDRKYTIIRKKTIRPDKVTLISGGGSGHEPAHAGFVGKGMLDAAVCGDVFASP
SQIQVYQAIRSTAGRKGTLLIIKNYSGDIMNFKNAAHLAGEDGIQVDFVKVDDDIAVEDSLYTVGRRGVAGTVLVHKIAG
AAAEAGMSLEQVKQAAQKAVDRTKSIGFALSSCTVPAKGSPTFSMEENEMEYGVGIHGEPGIRRESVLTADDLARRMVAE
LLQSLPEDEDNAPEVAVLVNGFGGTPLQELYLLNHSVQRELASRNVAVYKTLVGNYMTSIDMEGASLTLMKLDDELKNLL
RAECDTPALTVREYTEPLSWTAVAPDEEAYGDKPISFSVVTPEESAVIHNERLSLDNMIYLIDTMSEIIIENEVQFCELD
AHAGDGDFGMSVAKGFKQLKTEWNELTDRASDIGSFLDGCSLIIMEHCGGASGPIWGSAFRAAGRNVRGKAGLDATDIAG
MLQAAVKGIQDTGERSFGRGAVVGDKTLIDALVPYADSWEASAAAGTDLRSAGAAAAEAAVEGAKKTEAIVARMGRAGTV
GERSIGYPDAGAHALGVIFTGLAKALK
>lcl|NZ_CP026520.1_prot_WP_042226940.1_3753 [locus_tag=PC41400_RS18995] [protein=SRPBCC family protein] [protein_id=WP_042226940.1] [location=complement(4231324..4231749)] [gbkey=CDS]
MAKEISRIKIGAPAGNVWDALTRPELVKQWQYGSELVTDWQSGGEIRFRNEWEGQVFEQWGTVQEIVPDRLIRYTLFAPR
PGLEDKPENYFVMSYLLEDEGGWTTLTIEKTDNRPDAGSGVTEDEEVSPVLSALKNLVEGN
>lcl|NZ_CP026520.1_prot_WP_042226942.1_3754 [locus_tag=PC41400_RS19000] [protein=aminoglycoside phosphotransferase family protein] [protein_id=WP_042226942.1] [location=4232031..4232951] [gbkey=CDS]
MNKAQPYLEAIRQRYPELPLDTAEKIGAGQNNDVWLINGRLIFRFPKYESAIRDMEREVAFLLYAADHVSLEIPRPVFIH
TCPSLKELPFIGYNKIEGEPLEADLLQAEASNDSQEEHVPNEAFLRHIAGQLDDFLQELHGLDPKDLACAQREPGYDGYR
EWSDMYERIQSRLYPCMKQGARAWTDRHFSAFLGCKANFEITPSLIHGDFGTSNILYEPVSKRVSGILDFGSAGMGDPAV
DYAALLASYGESFFRLVLVKNPKAREMLSRIIFYKGTFALQEALFGVENEDPEALKSGLETLEALE
>lcl|NZ_CP026520.1_prot_WP_042226944.1_3755 [locus_tag=PC41400_RS19005] [protein=hypothetical protein] [protein_id=WP_042226944.1] [location=complement(4233075..4233539)] [gbkey=CDS]
MKKFRGKKRYFRHLWKKVNDFYLQPDNESWFDFWHFHLDFCGTGEHSLKLRREHIKAHLALYNILLKRLERIEKPSQSWI
CIHEQDAGLDAVYIHTPNPNGDNFPHKLKSIDWHYKLPNTFKDLIDLEQFNVGYYKYEDGNIYFIQSKEQDMKL
>lcl|NZ_CP026520.1_prot_WP_042226947.1_3756 [locus_tag=PC41400_RS19010] [protein=SgrR family transcriptional regulator] [protein_id=WP_042226947.1] [location=complement(4233614..4235455)] [gbkey=CDS]
MQTVLPFLELRLFFRDREPQKPFPVTVDELAGIWHCTPRYVKLIVRALCEQGWIDWQPGRGRGNKSKLTLVLDSDELLLE
EVRRRTEQGEVHEAMELMNRFGGPAVKDRFMDWISEGIGFSTAGVPDFPQETLCMPVFRSITTLDPGLVYYAFDCHMARQ
LFNTLVEYDEEQRHVIPCIAHSWESSEDAREWTFYLRKNVLFHHGRELTAEDVMFTLNRLRHDPGRYDSGWMFRDIEQMD
APDRKTVVIRLREPNYLFLKFLCTIAAAIVPEDVVRRDEAAFAEMPVGTGPFRVECRNQGRCVLEAFPAHFRGRAHLDRV
EVLIFPPEESGRLKEPDWTSIMTSHGETSGARREALVSGEGGWRNMETIFNCSSMLVFNQRKKGPHNHPLFREALHHIID
REALIADLKGERLCPAQGFHPQEDMGKRAGTYYPVKRRFEIETLLEQSGYAGEVLELVTYSHHEEDACWIQARARTFGVY
IEIRVRNAEEWSDRASLQRYDCQLYGIVLGGDEVRELEVYLQNNFFAALWDSATEEVVRRETAAILREPVEAARKERFAG
LERWVRQTHAVMFLVHKTTNTAFHQSVRGVKINSYGWVDFYHIWFNTGAAVST
>lcl|NZ_CP026520.1_prot_WP_042226948.1_3757 [locus_tag=PC41400_RS19015] [protein=MFS transporter] [protein_id=WP_042226948.1] [location=4235579..4236823] [gbkey=CDS]
MKTKKTKIFRIPSFHPVAWGVIIGTFLSRTGFFMTIPFLGIYLGKVKGIDPATIGAILAVSLFVGTLGSFAGGALSDRLG
RYPVMIASMAAWSLVFIGFAFADATWLFFVLSALNGLFRSVFEPVARALLADVTPAESRTEVFNARYFAINIGGAVGPLI
GLKLGIGGTSSLLPFLVSAAIYAVYAALLIVFMFSFRHMLQSNDEPGATVKQMMKIVFTDKVFLYLLLGNFFVAGAYSHL
DTTLSQYIGHDRVAVYSSLFVVNTLCVLLLQYPLTRFMKRYTSLTALKTGCLLFGAGLFGFGLFENTALLFLSMIVFTMG
EILCFVVGDVLIGEIAPERLRGAYYGASGFAFLGQSLCAWIGGLLLQGLGFGQGPLIFGILALLALAAFPFFHRGQQLGE
HPVGAEPSELSESC
>lcl|NZ_CP026520.1_prot_WP_042226950.1_3758 [locus_tag=PC41400_RS19020] [protein=NupC/NupG family nucleoside CNT transporter] [protein_id=WP_042226950.1] [location=complement(4237012..4238235)] [gbkey=CDS]
MSILWGLVGLAVILGIAYLMSDHKKKINYRTVIVGLGIQLVFGYIVLKWDTGKQMLEVVSAGFTNLFHYGYEGLSFVFGS
LADKTQATGNIFAIRVAMLAVFITPLIGILYRFGIMQFFMRVVGGGLGKLLKTSRAESLAAAGNIFLGLQEIPIMIKPYM
KFLTRSEMFAIMVGGLASVAGGIMVAYAALGIPIAYLLSASLMSAPAGLIIAKIMIPETDTPVEAGSRELEEEDSDTKDN
LINIISKGASAGMRMAVQIIAMLIAFVSMVALLNGLLGWVGSWFGYGDLSLQMILGWVFAPLAFVIGVPWSEATITGSFL
GQKFVMNEMIAFGSFAKEMEHLSPKTIAIVSFALCGFANIGSMGILLGSMRSLAPERSQEVGRLAFKAVIAASLANLLSG
TIAGMFM
>lcl|NZ_CP026520.1_prot_WP_241688697.1_3759 [gene=rbsK] [locus_tag=PC41400_RS19025] [protein=ribokinase] [protein_id=WP_241688697.1] [location=complement(4238240..4239130)] [gbkey=CDS]
MDSSLLLESLPHEGETLFASGKLVGPGGKGFNQAVSAARLGASVHFIGKVGADEYGKQLVSTLRQEGIHTEHVGVNPDLP
TGEALILFSNSGSNMIVVSAGSNMAITAQDIEESREIIQLSDVILSQFEVPIEVIQHVFTIAKEEGKVTVLNPAPAKAIP
PALLQVTDFLIPNESEMHMITGCDTNCEQDMVQGAATLLEQGVGCVIVTLGDRGSLICHPDGNLSIPAEKVAAIDTTAAG
DSFIGSFCTKLRRQERKQLRHVAEAVKFASRFSSIVVQRKGAFQSIPYAYELAEMS
>lcl|NZ_CP026520.1_prot_WP_042226954.1_3760 [gene=dcuC] [locus_tag=PC41400_RS19030] [protein=C4-dicarboxylate transporter DcuC] [protein_id=WP_042226954.1] [location=complement(4239232..4240590)] [gbkey=CDS]
MAELIFGLIVVVLAGWMISKKVNATVVLFVAGIVLLYGAVFMGHNVLPAEAATGSRWFDPMKSITNNFISNLGNIGMTIM
VLFGFSSYMTKIGANEVTVQTLMKPLAGIKSVYVLVPVIFVVGNLLSLVVPSASSLAILLMATLYPVLKRIGMSGLTAGA
VIATTATIIPTPLGADNVIAAQTLGIDLMEYVMKYHAVISIPTLILMTFAHYFWQKYMDKKNPETDTIDESKLGTLRTDL
PPTYYGFLPILPLVLITVFGIFVKSIKIGLVEITFISLIVAVLVEMIRKRKIQQVFEELMVFFRGMGDGLAMVVSLLVAA
SLLVDGLKALGIVDMLTDSVKDLNGAGMILMFAFSGITALIGLISGGGLAVFYASVGLLPAIAQNVGIDGAMLSIPMQLI
ANLVRSVSPVAACIIVVCSIIQKSPMEVIKRTSVPIIVGIISTLVLSYVMFS
>lcl|NZ_CP026520.1_prot_WP_042226955.1_3761 [gene=rihC] [locus_tag=PC41400_RS19035] [protein=ribonucleoside hydrolase RihC] [protein_id=WP_042226955.1] [location=complement(4240620..4241543)] [gbkey=CDS]
MSVRKTPLIIDTDPGIDDAVAIGAALHHESLDVKLLTTVAGNVSVDKTTQNALKLIEFFGTGTPVARGARQPLCMPYEDS
SHIHGESGMGGYEFPEPTTSIHPEHAVNAIRETIMRSEEKITLVPIGPLTNIALFLAMYPELKSRIERIVLMGGSASAGN
HTQTAEYNIYADPEAAKMVFASGLDITMVGLDVTSKATLTPDNVVKIRDLNETGNMLYSMFQHYRGGSLKTGLKMHDLCA
IAYIVKPELFKTEKYFVDVESAGVYTAGTTIVDQKNRYNKEPNVNVCLDIDVDAFRTWAIECIAKAK
>lcl|NZ_CP026520.1_prot_WP_084706517.1_3762 [locus_tag=PC41400_RS19040] [protein=LacI family DNA-binding transcriptional regulator] [protein_id=WP_084706517.1] [location=4241789..4242772] [gbkey=CDS]
MATIRDVSKLAGVSVATVSRVINQKGYVHEDTLQRVHKAIKLLNYVPNDVARSLSKKSTKTIALFVPDITNPFFNELALA
VEKVTQLYGYTTILCNTNESAKIERQYIQNLKQKYIDGFIMATNTLTEEEIDQLDIPMVAVDRSSTPVVPTVTSKNRDGA
HLAVQALLQQGCNKIAHIRGPKGLWIAEERCQGYLDLVENMPWFTPKLIANGNFQLKEAAEAAKKLFKEHPDIDGVFAGN
DLMAIGTLRALQEHGIRVPDQVQIIGYDGISLAEMVYPELSTVSQSIYEMGMLAARILIKKIEQIHLDEENHQLEVKLVQ
RGTTRHG
>lcl|NZ_CP026520.1_prot_WP_042226957.1_3763 [locus_tag=PC41400_RS19045] [protein=LysR family transcriptional regulator] [protein_id=WP_042226957.1] [location=complement(4243036..4243926)] [gbkey=CDS]
MTHSQLMLFVKIVETGSFTKAGEQLNMTQPAVSRAISALESELGVTLMIRDRKSGVILTDIGLRLLAIFRDILHGYNKVE
QEVTAEKGFEVGTVRVGVFPIVSAHFLPKILRVIGEKYPGLTFELHEGTIDEIKKWLDSRMIDVGWIIPPSNELETIPFF
RDELCLLHRDDHPLQSRPTIHITDLDNEPMIVCKGGFVTPIYELFQQFGATLHAKYDGHNISTALSMIQEGLGVSIVSKT
SLSLSALPPNVRVRTIDPQPFREIHLAVPSLKEAPHGVKLFLQTARELYLSGDRNH
>lcl|NZ_CP026520.1_prot_WP_042226958.1_3764 [locus_tag=PC41400_RS19050] [protein=DMT family transporter] [protein_id=WP_042226958.1] [location=4244026..4244955] [gbkey=CDS]
MQLSRTLTFILLTFLVMLWGVNWPLSKYALEFAPPLLFAGLRIFIGGLILLIFALPRYRKLNMRHTWNIYLISALLNIIL
FFGFQTFGLNMMPAGLFSTIVFMQPVLLGIGAWLWLGESMTGFKIIGLILGFIGVAVISINDGSGNISSAGILLGLASTI
TWTFGTIYMKKTSTRVDAIWLTTLQMIFGGIVLLVAGTTVENWTDIVWNQPFISTLSVIAIFCTALTWLVFFLLVRSGEA
GKIGSFNFLVPLIAIVISVLVLGETITPKMVVGLILIIIGVALVNLKLKSLQKRDKKHMGNMKARKDNV
>lcl|NZ_CP026520.1_prot_WP_042226959.1_3765 [locus_tag=PC41400_RS19060] [protein=hypothetical protein] [protein_id=WP_042226959.1] [location=complement(4245986..4246534)] [gbkey=CDS]
MKEPSEQLIEMLGHPYSPKNVQALLQSFGIKRMPPANSYFNDDIIWSVKSSIRIDLYRAPKINDLTGLNYTNEDGWIIGA
IHFLAPGSDDRIKTPFPGDLPKGITMRSTPDESIQAYGQPQLDEECEWPGFSGRILAWRKPGINIAITYADSEIGSSMIS
CTACLIGCIGAWRNDNPEVFAP
>lcl|NZ_CP026520.1_prot_WP_042226960.1_3766 [locus_tag=PC41400_RS19065] [protein=hypothetical protein] [protein_id=WP_042226960.1] [location=complement(4246574..4247050)] [gbkey=CDS]
MNPLDYIEYLGHSSVYPPLVDQLSRDKITWRPNVQRNLNTTYFVSGEGLVLHFDIGAENNGITKKSEGDYIFDELTMTIM
EENKKHGKYVGPIPYGLLQNDSRAQIEAKLGTPTRRAEELDNYYLDGLVWTAAFEGERFQFLKLAVPTNGKRKYGLCP
>lcl|NZ_CP026520.1_prot_WP_042226961.1_3767 [locus_tag=PC41400_RS19070] [protein=histidine phosphatase family protein] [protein_id=WP_042226961.1] [location=4247240..4247821] [gbkey=CDS]
MKTFVYMVRHGESPKTEGNERTRGLTLKGSSDAQIVAQLLKDEGIDAFISSPYKRAILTLEGLARSLGKEMIVIEELKET
VFIGDDKILPDTEVYPLVKKMFSDPDFSLPGGESFTNSQNRVVAILKNILSEYKGQKVAIGTHGAVMTMMMGYFDPQFDL
DFLLKTSKPDIYKMEFDEGILTTTARLWKVSPA
>lcl|NZ_CP026520.1_prot_WP_042226962.1_3768 [locus_tag=PC41400_RS19075] [protein=Cfr family 23S rRNA (adenine(2503)-C(8))-methyltransferase] [protein_id=WP_042226962.1] [location=complement(4248036..4249067)] [gbkey=CDS]
MRLPSKYETIRQILSDFKQPEYRYAQIMDAIFKHNVGEYGRMTMLPQFLRDELTRMLGPDVCSIVPVKELTSKQVNKVLF
AIAGDERVEAVRLTYERGWKSYCISTQCGCGFGCRFCATGTIGLKRNLTAEEITDQLLHFRLNGHALDSVSFMGMGEALA
NPYIFDAITILTDPQLFGLGHRRITVSTIGLLPGIDRLTREFPQVNLTFSLHSPFDDQRSELMPINDRFPIHDVLKALDH
HIRHTGRKVYIAYILLRGVNDSTAHAEAVAELLRGRGPVEHLYHVNLIPFNSTEVTPDNYQQSESNRIKAFVGILKSKGI
SVTVRTQFGSDINAACGQLYRSE
>lcl|NZ_CP026520.1_prot_WP_042226964.1_3769 [locus_tag=PC41400_RS19080] [protein=YdeI/OmpD-associated family protein] [protein_id=WP_042226964.1] [location=complement(4249607..4250176)] [gbkey=CDS]
MEMNHLILVKSREELRFWLQENRKTEKFCWVLVSMTPKSDTLLYLDAVEEALCFGWIDGVKKKIAETGLAQRLSPRSKKS
SWTELNKERVRRLEKLGLMTDEGRKVLPDMDDDSFRIDPVIEQRLKEEKQVYENFKAFPELYKRVRIDTIQSIKNQPELF
KSRLDKFITNTRENKMYGQWNDNGRLLDY
>lcl|NZ_CP026520.1_prot_WP_042226966.1_3770 [locus_tag=PC41400_RS19085] [protein=VOC family protein] [protein_id=WP_042226966.1] [location=complement(4250194..4250658)] [gbkey=CDS]
MAAIGYVNLDGNAEQAIEFYSEALSAIEVKKVKFGDIPQDPNYPMPENELTMIMESSIEFAGGKIMMSDILPSMKAVTGE
LVKGNNILISLVMDDKQKLEEYFTHLSVGGHVIMPLSDTTWSSCFGMLVDKFGVSWKFNSDADKFLDNVIANKQ
>lcl|NZ_CP026520.1_prot_WP_042226968.1_3771 [locus_tag=PC41400_RS19090] [protein=YafY family protein] [protein_id=WP_042226968.1] [location=4250758..4251669] [gbkey=CDS]
MDKVERLISIIMILLKKNIVSTNEFAQLFNVSKRTILRDMETLSLSNIPLYSIHGVNGGYGIMDEYKVDKRLLSSSDLEN
ILAALGGLEQILISEEVELTIKKIEAMVSPLAPKSSIQLSFYDWEGRSEIIQTLKTCQESILKRKLVSFEYIDKNGTVTN
RIVEPYQLHFSETSWYLKGFCLHRMSNRTFKLSRIDHLTMNEQTFSPRDDLLEQEQEASYQPELVAIQALISPSIKDQFI
ERYGRKSIETYNSEFLLATIDVPQNSTGFQFLASFGTNLKIVEPKTYVEDFRTYLNKMVDVYS
>lcl|NZ_CP026520.1_prot_WP_042226970.1_3772 [locus_tag=PC41400_RS30610] [protein=flagellin] [protein_id=WP_042226970.1] [location=4251799..4253601] [gbkey=CDS]
MKISHNIPAYRINNTLQKNSKTSSSSLEKLSSGLRINKASDDAAGLAISEKMRAQIRGLYQASKNVMDGISLIQTAEGGL
QEITDILQRQRELTIQGLNDPLTDEDRRKIDMEIRQLTDEIESLSKKTAFNTMNLLARTDYKILADRSSSSSTFSASDPP
PVSSINKTIVYKPRGTPDEARHVIGSSDTSETTYTFSNTNNISNIVLPDGTEAFNEYNVDTLTTTKTDTNKTIYESLTAV
NDPEFSKPAYWYSHGMNKTFFGPRHFGEAYGTMLEDIEVNGRSWPIEYSSRSNSGSDPAWDHMWFPGTNLSIMRYRTLLA
DNSMEIRYEITNKDSFNTNINLSNIVNPPANSVITGAGGSPLANGHTVISPPGGSSFTMTGTEADAGIQFPLGSLSPTEL
SIDNPAAGQPQVNFNWQLNVPSGSSVTFGFKYGPFSLNYEPFEQAQETVETKHIETTVNTDITDIDYLPPKLTIQAGANK
DETIVIPLFNVGTQGLGIKDLGLLPPSIPEQALAKADGAIEKVLHYRAMYGALQNRMEHTLNNVNNSAENQMTAESRIRD
ADMAKEISNMTKSNMLTQAAQTILAQANQNPQAVLQLLKP
>lcl|NZ_CP026520.1_prot_WP_042226971.1_3773 [locus_tag=PC41400_RS19100] [protein=MerR family transcriptional regulator] [protein_id=WP_042226971.1] [location=4253972..4254790] [gbkey=CDS]
MFKISEFSRLSKVSLKTLRYYDQIGILKPRKVDHDTGYRYYSADQLLELNRILIYKELGFTLPQITQLLQEDITLENIQG
MFKLKRSEIQQIIDTEQAKLARIEERMQLIEREGQVETGQEIRIKAEDARQFLFLKAHGREEDISDLFRTFNQLLTKETR
QLVQGPQVVLWKEIDGKEDEFEFEIGYFLTCEPLLSPDPFQLRTLPPEPMMATMAFRSDSNFACTACVDLAKWIEKNNYQ
IKENEPGREIYLPLSTKQDAQLMEIQIPIIHR
>lcl|NZ_CP026520.1_prot_WP_042226973.1_3774 [locus_tag=PC41400_RS19105] [protein=MFS transporter] [protein_id=WP_042226973.1] [location=4254812..4255987] [gbkey=CDS]
MKNNRIIYVLALAVFLIGTIEYIITGVIEMIAADLGVSTSEAGLLVTVFALAAAIVAPILIALTINVDRKKLLMAALSVF
IASNGLMFVNLSYETVLWIRIIQGASGGITTVVAMAVATRLVEKERRGNAIGIILMGLSSSLVLGVPIGTFFSEMFGWRV
LFIGIGLLSVLPLLIIYKKVPAIKEEDKVTLRMQLSILKNSNILTALVITLFYIGGYSTLFTYMTPFLQAGSSLSITEIS
GVLFLAGICSFVGSKVGGQLADAKGPKFTICLGLLLQGATLLLFTLAGVNLFVLILVIMIFMLATWSISPAQQLYLVTLA
PRNPDIALSVNTSFIQFGFALGSALGGLVISRTSVLYLNWLGLAAVSIALLLAMLLFKKMRSGIGTPVVTK
>lcl|NZ_CP026520.1_prot_WP_042226974.1_3775 [locus_tag=PC41400_RS19110] [protein=hypothetical protein] [protein_id=WP_042226974.1] [location=complement(4256200..4257249)] [gbkey=CDS]
MKKIVLIVFFICIFALSFYYYSLNGDKMNSKVRNVMINEINVLENEHVLLSNTASFSNSTFSFAQRLRLDPNMEVKYDLR
DDDYYRYYINKEEKDIISGEYTIDTIDDANEIILVAIQGNKVAPLRIKNDNKWNNSLRINFAANSTTKIPVEIKWNPNKE
EDLIVFPIKKDKHGEYYSGGNLGVVRYYIQNTNSPTISKDEIHKQSVINEKLVSQEGIKPVPTLTWIGDDGNPIKLVLDK
NQMMLSNTKSYSLKISATPYSTVSDLLLLNENFEIKVIKRNIQMYEGRETNLTFPPEIFNKTHENSRQFLLLLNNRGRSM
LTDMMSINEMNKAFPTSFQQVIEVFPLKN
>lcl|NZ_CP026520.1_prot_WP_129112512.1_3776 [locus_tag=PC41400_RS19115] [protein=hypothetical protein] [protein_id=WP_129112512.1] [location=complement(4257320..4257880)] [gbkey=CDS]
MKQKLLKIVSIASVVTALSIPGSVFANSKEDSKAIEENLKIITSYSPEITYDIVPKTGTITAPSGNNEISPSSLSFVQVA
TGTTETDYISAVKHVMTSGSTKGKKLTVTTSATASIWFEGNFTNGLLNTGPKDMAIGYGTATSITGSTTLVDNTVVKGSY
RGITYHSCVYDFALYELKSSESVTVY
>lcl|NZ_CP026520.1_prot_WP_042226976.1_3777 [locus_tag=PC41400_RS19120] [protein=demethylmenaquinone methyltransferase] [protein_id=WP_042226976.1] [location=complement(4258327..4259031)] [gbkey=CDS]
MSEQTKMERVHDVFEKISNQYDSMNSVITFKRHIAWRQDVMKQMNVREGSKALDVCTGTGDWAIALANVIGTHGEVIGLD
FSKNMLKIAQDKQEQQALKQLTFIQGNAMELPFEDHSFDYVTIGFGLRNVPDYSTVLREMYRVTKPGGKVVCLETSQPTL
IGYRQLSLFYVRYIMPLLGKFIAKSYDEYSWLQESARNFPDKNVLKQMFLEAGFSKVDAKSYSGGVAAMHMAIK
>lcl|NZ_CP026520.1_prot_WP_042226978.1_3778 [locus_tag=PC41400_RS19125] [protein=hypothetical protein] [protein_id=WP_042226978.1] [location=complement(4259095..4259664)] [gbkey=CDS]
MSWSKLKQNLESFLCPALDGRVEYRATGYRYLPDKAGICYITVDKKNVLNMNDATNLIRWYQTELEIKNDPDIQIPISHE
EMEAVRQETKGKVPEDRLKVIVRSRKSSEHAKELLTAQTSLSKSNFIVVANKFLSTSIEESIESHDILLNILALVDRRVG
KKRILNMSEKMKLKHPIVQYFYELRRSTL
>lcl|NZ_CP026520.1_prot_WP_042226979.1_3779 [locus_tag=PC41400_RS19130] [protein=nucleotidyl transferase AbiEii/AbiGii toxin family protein] [protein_id=WP_042226979.1] [location=complement(4259955..4260851)] [gbkey=CDS]
MIKFENLKDMNDEDRELIVLEALLRRISLVNMPFVLKGSLLTRQYLNNPDTRHVEDLDFLYMDKIKDVEHAHETFTNWMI
QVTETDLGDGITFRSFRENAFWRCIDYAMSDDFPTINTDLAYFFTDEPREVNHYSEIYLDVSFNLEMAVDPVSIEYRPVL
GDTFSVSYSVPLSIQIAWKLHQTIVRPRFKDLYDLKYLLSHPSYDECALNETLQALVNECSIDQSITPIAIKKILVDDLG
GLYHSLSNDYSLKKYAGSIIQEVYFMKLVADLRKIMNQVGINENAYVNLPRPTHRTHQ
>lcl|NZ_CP026520.1_prot_WP_042226980.1_3780 [locus_tag=PC41400_RS19135] [protein=hypothetical protein] [protein_id=WP_042226980.1] [location=complement(4260876..4261460)] [gbkey=CDS]
MEFEFHITVNNLKLVEKEFFVNTCILEQVKPLMIVLDKGDYINQPMCTGIIESTDCYEAKKEIEKIATKFQKNGFTIMRT
KIEISPNEEKYFRMPVTRNSKPYFEWHGKVAVKEVARLKNLCEGLGGHISRNSLATNGEIRFITVREYESKEKFYARVYK
LHSILQENKIDVLKEQYELCIYDSREELDSGWIS
>lcl|NZ_CP026520.1_prot_WP_042226982.1_3781 [locus_tag=PC41400_RS19140] [protein=SMI1/KNR4 family protein] [protein_id=WP_042226982.1] [location=complement(4261636..4262052)] [gbkey=CDS]
MRYTNIINSIRNNSSRHNFEVLSEDEIKRELEVEGIPKEYIDFLREVGYGTVNGTYFNFYSGLVEVDEILGHLFDEDSHP
ELKDVLLFGDNFSGDAVGFLTTDNWAIVEIWHDDDLSINPTEEKTFEEFVVTMFEKFN
>lcl|NZ_CP026520.1_prot_WP_042227434.1_3782 [locus_tag=PC41400_RS19145] [protein=ATP-grasp domain-containing protein] [protein_id=WP_042227434.1] [location=complement(4262107..4262835)] [gbkey=CDS]
MKVYIRKNFDNEIANHNFYAAYDGFKQMGFEIHYFKNIDELSDNKKEDIIVSYVDDVRTVLFKHDIIPPEIDYPEELKNY
LGRKVWNSKLSVIANNPENWNVFIKPVEDKKFTGVVVRSTKDLIGCGTNGEDPDILCSEVVNFVAEWRCFVRYGKILDIR
RYKGNWRVHFDYKVIENVVSQFESAPNGYAVDFGLTDKGETLLIEVNDGYSLGHYGLVSLDYAKLLSARWSELTSTIDEC
DF
>lcl|NZ_CP026520.1_prot_WP_042226984.1_3783 [locus_tag=PC41400_RS19150] [protein=hypothetical protein] [protein_id=WP_042226984.1] [location=complement(4263193..4264659)] [gbkey=CDS]
MKYPSFIHDKLNELENVIASWPVNIEIDHVIKWILQFDPNDYYIALRVINNLNVIGYDDMNSALQVAYSRLLRMAKDKNT
VIKGDNTLFAGIGDAGKSGSMISYHFRLINDVSEENFLDEDSFHHLEKERVNNIVLLDDVISTGNQATREINVLTEKVTP
FKVKNIFLLTVCGMRDGITKITDETKALTFSAFEYGKEDTVESLDSKFYDGVLHDSRKELRERIEYYGKISSNSNNGMGY
GGIGGLIVFPYNTPNSTLPIIWSDRNGWIPLFKRVRRVNGISSYYKQMDKAIETKVNQEKPQVIGQETEPTNLNLYVEGK
TDEDFFNVLTEEYGLAERLKVEKLNVISLGGAFFSEKLIKILSDLGEKNLFLFENDELFYTGSRGIKKYVEEKFPQLPVI
YLEPSIMQFIRINDLIQDGDIALSLEKTLGESISLEHIHSDPRLLNEIEKHLRTKYRNRLKPIFSRFIDEERVAIFIEQV
NQKLYGKS
>lcl|NZ_CP026520.1_prot_WP_042226986.1_3784 [locus_tag=PC41400_RS19155] [protein=hypothetical protein] [protein_id=WP_042226986.1] [location=complement(4264725..4266581)] [gbkey=CDS]
MTTKGFFLTKLCLLGKNELMTEIEFKKGLNVIMGPTNTGKSFLFECINFMLGGKDAPEQIDEIVDYEYHTIFLELESNDG
LKFTFERQLSGGNFKKYNCPIHELTTESKFQNLSQKHSEKKESMSKFLMSLTGYKNTNLFIKTNKNNKLRRFTYRNYNDF
ILIDEVKIIAKESPVHSDNNKTKTAEEAAFRLILTNKDDSELSENVVASNSGSVYTAQVGLIDRLINDLELNISNLTPLQ
SEVDLDAIIKGLTLKRSGISSEIEKLTDSRKKLWKNIQQDESKLLSTYELLKRFLLLKDQYETDMQRINFLIEGDHFFSL
LNYEKCPHCNQTLTLNNGELACSHLEIEQKKESYKIELQKLLFHLEDLNSTIESMNEEVSSINTKLVDDKEDYNKLSKVL
DEQLEPEILIIEEELQGILKNQKTINEVEQKKHTLNKLLDEKRNILDLIGSEPDTGATSEQLLDYTKFYDDLCESIKYFL
TGWKYPGFESVKFDVKQKDILISGKARRLFGKGYRSISYSAFVLGVMKYCNLKDLPHPGIVILDSPITSYKEEDGDEDKT
SDDLQSRFFEFLAEVTLNNQVIILENKKPSEEVIKEINFIEFTKSHSRGRYGFIRTSN
>lcl|NZ_CP026520.1_prot_WP_042226988.1_3785 [locus_tag=PC41400_RS19160] [protein=hypothetical protein] [protein_id=WP_042226988.1] [location=complement(4266578..4267090)] [gbkey=CDS]
MTINNETTESSSINSSFEIGLRALTILAVTSPVSLDLQRLIYYDYLVIHTKDVGAVDSPPSLHPDTPHRSGGIIVRRKAM
QKGLELMYSKSLINIIFDEKGISYSASELTKPFLDLFESTYSKKLQNNALWVVGYFSGYSEEEMKFFIERNIDNWGGEFM
YEAFVRGGIE
>lcl|NZ_CP026520.1_prot_WP_053228641.1_3786 [locus_tag=PC41400_RS19165] [protein=hypothetical protein] [protein_id=WP_053228641.1] [location=complement(4267090..4268196)] [gbkey=CDS]
MIELENNMIDINEPEVKTKLSNTDLLLGKAIEPIKRLQVISDKDFEDLVREWSCDYLKDKYVKVRRCGAAGDMGRDVIAY
VEYEKGKKLVWDNYQCKHYNSPLSPSKIWIELGKLCYYTYKKEYSVPRKYYFVTPKGISTTLSNLIDDPHKLKKQLISEW
NDKCRSKITSGANIELTEDLLEYVKNFDFSIIDDIDPQELIEQHSQTKYFYYRFGGIHKTRPEPVNPPENISASELLYVR
KLLEAYSDNHKSTIRDTTELSEHYKYNSHFNRQRKCFYSAESLMKFERDTLPPGVNAFEDLKQEVYDGVIDIIESEHTDG
FERVKEVCKAARSLNMPSYPLYNSIKGNDLNGLCHHLANEDKISWVEQ
>lcl|NZ_CP026520.1_prot_PC41400_RS19170_3787 [locus_tag=PC41400_RS19170] [protein=tyrosine-type recombinase/integrase] [pseudo=true] [location=complement(4268493..4269314)] [gbkey=CDS]
MLDQFINEGLRGKSEATVKTYRHALQQFSQWLDGCGTTLEHFSRTDVQQYVTYMASKKKTAATINKVYNAIKKFCRWAKK
LDCIEEIHIVKAPDYKQTAPKVLDRLERKRLEREVDRSGNKRNYAIIKTLLHTGLRVSEIVALDRSDVDLSERKGELRVR
A*KGNKARTLPLDVDTRWAIRKYLEERSDDNEALFISNRDKRISVRCVQHLLEQHDLHAHQLRHTLITGLVRANEDIAVI
QSLSGHSSADMILRYSKPTEEDKLKAIEGLYNE
>lcl|NZ_CP026520.1_prot_WP_042226989.1_3788 [locus_tag=PC41400_RS19175] [protein=recombinase family protein] [protein_id=WP_042226989.1] [location=4269566..4270969] [gbkey=CDS]
MVDVYADEGITGRSMARRESLKRLIHDAEKGLFDVVLIKALSRLSRDTPDSINIVRTLHSHKVEVFSSREGMIDDELILS
VISAVNQRQSEDTSYNISWGIAVKSKEGIFHGTPPIGYDKVKPGKLVPSVVHAPTVQLIFDLYLNIGMGVQAIANHLNDE
LGVLPSRKNSKKWHDSSIRLILTNRHYTGDLVQGRSKVDSKDKKFLQDNGYKKRHKIDESEWIVVPNHHEPLVTREQFEA
VQEKMKQKAARMFRGRGRKSLFARVAFCVDCGAGMNFRNDRKSYVCATYQKNTSKKCSSHIIKHDALKQVVLSHLKDLLE
NSLNMKSLVDVTIQRAGIKKNNAVKELEKLQREISKLDKELVQLTRKWTSGDIDTDMFNACSVAVKNERQALEERAARLQ
TEIAHEHDTESRVSAFKAELMKFASLDVSNEEILRDMIHKMINCVDVHEDGSIEISYNFQNPLKQEA
>lcl|NZ_CP026520.1_prot_WP_084706518.1_3789 [locus_tag=PC41400_RS19180] [protein=RNA methyltransferase] [protein_id=WP_084706518.1] [location=complement(4270971..4271210)] [gbkey=CDS]
MSLSLTRRGGCEQSLLQAILQARPARLVYVSCNPATLAKDCETLLRGGYQIEWVQPVNMFPQMTHVECVILMEWRGEPK
>lcl|NZ_CP026520.1_prot_WP_129112513.1_3790 [locus_tag=PC41400_RS19185] [protein=hypothetical protein] [protein_id=WP_129112513.1] [location=complement(4271176..4271436)] [gbkey=CDS]
MTELQIPAYNERTREGMVWTVVARVGEVTGHVQLTAYTHNRYGLAAGFQAVCAAYSFEGCAEKLLPEWVGRCIKPDVVVV
DPPWRL
>lcl|NZ_CP026520.1_prot_WP_042226992.1_3791 [locus_tag=PC41400_RS19190] [protein=MFS transporter] [protein_id=WP_042226992.1] [location=complement(4271627..4272874)] [gbkey=CDS]
MATKTIQPASIAAKTIYPILLTISMMHLLNDTIQFIVPSLFPILKSSLSLNFAQIGLISLMINLTAAIMQPLIGIYTDNK
AKPMMLPFGMLFTMLGVITLAYAHQFWVVMFAVMLIGLGSAIFHPESARVSFLAAGDKKGAASSIFQFGGFMGQAIAPVL
TAVWFVVTGQQGLVWLSIVIAVGFFAQFYVARWYQNQLVPADAAGAKTKQSQVSPTELSTGRIVLAITILMILVFSKNLY
VAAISSYYSFFVIDRFGVTMGQAQIILFVFLAANVVGLLLGGLLADRFSRLKLIWFSILGTAPFALMLPYSNLTFSIIWI
FLAGTILSSSFSVIMVYANELLPGKVGLVSGIFFGLAFGLGGVGSAILGNMADTTSIDFIIQCCAYLPLLGLLTIFLPSD
KKVKKEAQQVSVAQS
>lcl|NZ_CP026520.1_prot_WP_255246823.1_3792 [locus_tag=PC41400_RS30615] [protein=hypothetical protein] [protein_id=WP_255246823.1] [location=complement(4273308..4273433)] [gbkey=CDS]
MKEKREKALFLSQKKIASRRYAEYNMQYITTKICSCIAQTT
>lcl|NZ_CP026520.1_prot_WP_042226993.1_3793 [locus_tag=PC41400_RS19195] [protein=GntR family transcriptional regulator] [protein_id=WP_042226993.1] [location=4273567..4274238] [gbkey=CDS]
MPIPDNYSTPVRSTAKDRTLSQLQKWIIDGTLQPDEKLNDAELAAALGVSRTPVREALQLLEAQGLVQMFPGKDTRVTRI
DKDDIYKPYAPLVVLHALAAETAAQLIQPEQIEQLRELNQAFTKCILEKDLYQALEYDEEFHNLIVEIADNSYITSFCSS
LQLHIRRFKYVFLNHPIEQTQTAVQEHEQIIQALQDRDSATAARIIKQNLSRPLDEFHKQLHE
>lcl|NZ_CP026520.1_prot_WP_042226994.1_3794 [locus_tag=PC41400_RS19200] [protein=hypothetical protein] [protein_id=WP_042226994.1] [location=4274607..4274879] [gbkey=CDS]
MHFTPFEEGHLPEVLVIYNYYVANTTVSFHTETLTPDERRSQVLSGDKRYRNYVIEDRNTTLGYEKCAHFKEIGRKFDRW
PDIASYQKLI
>lcl|NZ_CP026520.1_prot_WP_042226995.1_3795 [locus_tag=PC41400_RS19205] [protein=MFS transporter] [protein_id=WP_042226995.1] [location=complement(4274925..4276112)] [gbkey=CDS]
MKGALSWPFLRLYMLTLLYFSANSILNVIIPLKGESLGATNTTIGIIMGAYLFTTMFFRPWAGQVIHKHGPIKVLRTILV
INGLALLLYTFTGLGGYFAARMLQGVCTAFFSMALQLGIIDALPDKDRSQGISMYSLCASMPNIIGPLLALGMWQTADSR
YFIAAMITIAILTGVVGYSAGMEKKEAEPAAGQPGHGAKMLHSFGHLVKNPHLFKCSVLMLTASLVFGAVTTFIPLYAMQ
IENGNAAMYLMIQATAVVAARFALRKKIPSDGKWHSSFMMVIMLILAAGAQCVSFSAAGGTVFFYVGAGLTGIAQALLYP
TLTTYLTFVLPQHDRNVLLGLFIATADLGVSLGGVVMGPVADLSSYSFMYLICAILGVSMIFFAYDRRKIFVGSR
>lcl|NZ_CP026520.1_prot_WP_042226997.1_3796 [locus_tag=PC41400_RS19210] [protein=ABC transporter ATP-binding protein] [protein_id=WP_042226997.1] [location=complement(4276155..4276898)] [gbkey=CDS]
MLRIEGVEKSYKTGGLFSKKKQNVLTDISFECGQGECLGIIGESGSGKSTLGRLIIGIEEPDQGTVLFEGKRIRERNARR
GNISAVFQDYTSSINPFYTVEQAMMEPLKLQNKARKEASGKIDLLLRQVGLDSSYRTKYPHELSGGEAQRVCIARAVSTE
PKCILLDEAISSLDGSVQIQVLQLLKRLREIYNMGYIFITHDIQAAAYICDRVVFMRQGRIEETVNIAHLKDVQSAYAKK
LLQMIIT
>lcl|NZ_CP026520.1_prot_WP_042226999.1_3797 [locus_tag=PC41400_RS19215] [protein=ABC transporter ATP-binding protein] [protein_id=WP_042226999.1] [location=complement(4276885..4277706)] [gbkey=CDS]
MNSLEVKNLKVWDNLTGEVIIPNSSFHVKKGSCLAIVGESGSGKSLTCRAVMRLTKANIRQSGDILFDGVNLSELPEKEM
RKRRGKQLCMILQNGMRAFDPSCVVGVHLKETLAGHFGWSRDEITAKMAKAMESVMLQDPIAVMNKYPHQLSGGMLQRVM
IALAIVLEPDLVIADEPTTALDTISQYEIVEQLAQLRKRMGCSMIFISHDLGIVRKISDEVLVMKEGEVVERGSTAAIFS
EARHEYTRYLVSSKLSLNHHFRNLMGGVSRAED
>lcl|NZ_CP026520.1_prot_WP_042227001.1_3798 [locus_tag=PC41400_RS19220] [protein=ABC transporter permease] [protein_id=WP_042227001.1] [location=complement(4277714..4278607)] [gbkey=CDS]
MKRFKFIWKDKPAVLSLFFIAAILAAGIFAPLIAPHAPGEVRMDLRFAGPSPDYLLGNDHLGRCVLSRLIYGIRPSVLWV
MAALILTVGIGAFFGFIAGYFKGRTDAVVMRICDVMLSFPGYVMILAVIGIVGAGFENILIAFVCLKWAWFARVIRTSVM
QVAEADYVKFSKALGAGSFQIMIKHILPVSLSDIAVLASSSFGSMILQLSGFSFLGLGVQAPNAEWGMMLNEAREVMFSK
PGLMLAPGLAIVLVVSAVNFLSDALQAALDPKLRTSKQNPQGQTIASGTAIQGNGVA
>lcl|NZ_CP026520.1_prot_WP_042227002.1_3799 [locus_tag=PC41400_RS19225] [protein=ABC transporter permease] [protein_id=WP_042227002.1] [location=complement(4278604..4279539)] [gbkey=CDS]
MGSYILKRCLLSLPLLAIISFLTFVLNHLSPLDPAEVVLHAQGVPNITEELIAETNRALGLDQPFLIRYFNWLVSCLQLD
FGDSYVTGKPVWSLLGPAFLNTLKLTLVSMAAIVVLSIALGVLCAWKEGKRVDKSVRGVSFLLTSMPPYWLASLMVWYFS
VHLDLLPTSGMDSFKSYLLPVAVITISYTGIYFRIVRSSMIGQLNEDYVLYGRASGLPERIITMHILRNSLQVAISVFCM
AIPIILGSTVVVENLFAWPGLGTLSVKSILSRDFPIIQAYVLVLAAAFVLFNTLSDMILAAMNPKLRKGMS
>lcl|NZ_CP026520.1_prot_WP_042227004.1_3800 [gene=nikA] [locus_tag=PC41400_RS19230] [protein=nickel ABC transporter substrate-binding protein] [protein_id=WP_042227004.1] [location=complement(4279594..4281198)] [gbkey=CDS]
MTRRIAGLLAALLISIALLAGCAGDKSAKTGAPEQSKELVYATAKDINDMNPHLYTGSMPAQGMVYESLVENTENGIKPL
LAESWETSKDGKTYTFHLRKDVTFHDGEPFNAEAVKKNIEAVQKNASKHSWIKLSTKIVSCTVRDEFTVELILSEPYYPA
LQELSMTRPYVFISPKNFVNGETKDGVSGYNGTGPYKLTEHKTEQYATFEANESYWGGAPKVKKITAKVLPAGEMTLLAL
QKGEVNFIFTDDRGADSIDVEAMNRLVQSGDFQLVRSKAMNTKMIVANSSKSGPVHETAVREAIWYSIDRETVSKQIFNG
METAANALFSSNVPYADVDLKKRGYDPETAKKILEEAGWRLESGKEIRTKDGTPLTMKLYYDVNSSSQKTQAEFLQHTLT
TIGAKLDLIGEESSSIAKRRATGDYDLLFNQTWGLAYDPQSTIAAFTSESSYLHTTKGIPKAAELYKKIDEVMVSADEKT
RKSLYADILTLVHDEGVFIPVTNGSVTVLAPKNLKGISFKQTQYELPFEQMYFE
>lcl|NZ_CP026520.1_prot_WP_042227005.1_3801 [locus_tag=PC41400_RS19235] [protein=opine metallophore biosynthesis dehydrogenase] [protein_id=WP_042227005.1] [location=complement(4281224..4282546)] [gbkey=CDS]
MDDFQKVLVLGTGPASIQLAVLLKKVANCRVGIAGRESARSEHFFAALKESSRRIRVDIQNEKHRQLQGECVVDDVFHGY
ESITGKWDTVILSVTADAYIGVLKQIEEKLLQQVRCVVLISPTFGSNSLIRHFMTDIRADAEIISFSTYLGDTRWRHDKP
SDQVITTGIKKKVFIGSTKAPAESDNVAKLRALYEQLGITLEIMRSPFEAETRNISLYVHPPLFMNDFSLNVLFGEAAPK
KFVYKMFPEGPITPALIRDMVACWKEMTAILRKLHIDGLNLLKFMTDDNYPVRPESLSRHEIENFQHFETIHQEYLVYIR
YTSLLIDPFSEPDPNGKYFDFSAVPIRQIFVNQEGYWDIPRMPKEDYYRIKIIQGIAKYVDSSCPTIDTFISTYERKIEE
AAQHRLQGELLSDAFIVQSFEEDLKRIRIEIETGLVSKDI
>lcl|NZ_CP026520.1_prot_WP_042227007.1_3802 [locus_tag=PC41400_RS19240] [protein=class I SAM-dependent methyltransferase] [protein_id=WP_042227007.1] [location=complement(4282572..4283366)] [gbkey=CDS]
MHTPEQFLQTLNGFQAQFAQLAERYEGTAQGSSELEAILDDYSAFITNEANEESWEHLNRQDSGHMSRLVEDLRKTSAFC
VAIMEKYRAWKLLNGDAERTEYFRNIESCIEKEFGSFQVTADSKVLLVGSGSFPMTPLYIAKRTGAEVIGIDIDREAVEL
GRRVVEELGSGLKIRLENVFVEHLDDIREVTHIIFSSTVANKYELLDQLEALTERDVIVAMRYGDGLKSLFNYPMQKVNG
RKWSLAETVLRPGHVFDIALYRKA
>lcl|NZ_CP026520.1_prot_WP_042227008.1_3803 [locus_tag=PC41400_RS19245] [protein=diaminopimelate epimerase] [protein_id=WP_042227008.1] [location=complement(4283394..4284227)] [gbkey=CDS]
MKREIDFVKCNPTQNMTILVRTNHLAEDYMPIAAKMMSYDSVYAEQVGFIEKPGSYEADANLRMAGGEFCGNACMSLAAF
LASEKGMNPDDFMEIVLEASGTEQLITCHVKKVKEEYVCRLAMPVPKKIDRKTIKFEGDDLDIVTVRYPDFLHIVIEVDH
FNETVRNRAQALAKLLGAASEASVIGILLFKSDSDELAPLIYVPLLDSMIWERGCGSGTASLGAFLSWKNKGAVSAQIKQ
PGGTMTVMAEWKEDEVTSLTIEGAVEIVAQGRAFIDV
>lcl|NZ_CP026520.1_prot_WP_042227010.1_3804 [locus_tag=PC41400_RS19250] [protein=sugar O-acetyltransferase] [protein_id=WP_042227010.1] [location=complement(4284525..4285127)] [gbkey=CDS]
MTDPKPLTEKQKMLAGKPYKAFGEELLLERQHAKEVLFDYNALRPGEVEKRNALLRPLLGRTGKVFFIEPPFRCDYGYNI
SLGENFYANYNLTVLDCNRVTIGNNVLIGPNVSVFTAGHAIHPELRVNEVEYALPISIGNNVWIGGGAILNPGVSIGDNT
IIGSGSVVTKDIPADVIAAGNPCRVLRPITEDDRNQFRDA
>lcl|NZ_CP026520.1_prot_WP_042227011.1_3805 [locus_tag=PC41400_RS19255] [protein=PQQ-dependent sugar dehydrogenase] [protein_id=WP_042227011.1] [location=4285359..4287599] [gbkey=CDS]
MIMHCKQVALSLCFCLLLVLIPVPGTAKAESVPAPWKQQNIGSPALAGSASYDPGAGRFEVSGSGTDIWGKSDQFNFVYQ
PWTGDGSIVALVSSQTNTHDFAKAGITIRETLKPDSKHADLLLTPSNGFTFQYRTETGGTSESEKIASTSPAWVKLERKG
NVIRGYVSNNGLNWGDLGTLTLKMNASVFVGLAVSSHNTSKLSTAAFTNVTVNAAGDVFPPTEPTNVQARSVTDTSVEIA
WTASLDDVGVTGYDLYKDDVLTQANVAGTSYTFTGLKPATTYRFTVKARDAAGNSSAASAPLSVTTTSGSDTESPAAPAG
LASSSKTSTSVQLTWTAATDNVGVYAYDIFTNGKLSGSTDKTSFNVTGLAANTSYSFTVKARDLAGNVSPASKALSVKTN
PASSEPYTPEVVASNLETPWAVDFAPDGRIFFTERNSGRIRVVVNGQLKSAPVITLGSPFYYMPKSEGGLLGLALDPDFT
NNHYMYIYHSYKTSDNKVANRVVRLIESNNTAKIDKVLITNLPGAVYHNGGRLKIGPDKKLYFSDGNYGNKDDTLTYLGG
KIFRLNLDGTIPSDNPFGASSPIYSRGHRNPQGLAWQPGTNRLFESEHGESSKDEINFIQPGAHYGWPQYEGGNQNQPGI
TPPLLYASGTTWAPSGIAFVTKGPWAGDLLVTNLKGKQIIRMTVKEQNGKPVIDSGSLNYLYVNKYGRIRDIVEAPDGSL
YFVTSNNGDKNGDGTPDKLVRLAPNF
>lcl|NZ_CP026520.1_prot_PC41400_RS19260_3806 [locus_tag=PC41400_RS19260] [protein=tyrosine-type recombinase/integrase] [pseudo=true] [partial=5'] [location=complement(4287852..>4288055)] [gbkey=CDS]
KYKLGTITTHGFRHTHCSLLFEAGATLKEVQDRLGHSDIKTTMDTYTHVTKEAKEEAILKFASYLET
>lcl|NZ_CP026520.1_prot_WP_042227013.1_3807 [locus_tag=PC41400_RS19265] [protein=CatB-related O-acetyltransferase] [protein_id=WP_042227013.1] [location=complement(4288133..4288792)] [gbkey=CDS]
MNQPLFNHWSEIKYLKDIVTNPLIEVGEYSYYSGYYGNQNFEDGCVRYLWGDFKSQELFNPIEQYGWHLDKLIIGNYVCI
ASGVVILMGGNHNHHSEWATVYPFVEQVEQSYEPKGDTVIKSDAWIGMNAMIMPGVTIGEGAIIAAGSVVSKDVPPYTIV
GGNPAKEIKKRFTDSEINKLMEMRWFDWDRKMIERAIPLLSNSSIDKLYDFYNTEVKKK
>lcl|NZ_CP026520.1_prot_WP_042227015.1_3808 [locus_tag=PC41400_RS19270] [protein=DUF4184 family protein] [protein_id=WP_042227015.1] [location=complement(4289117..4289860)] [gbkey=CDS]
MPFTFAHPMAVMPWWNKTTLPVSALIIGCMSPDFEYFVRFRAAGVWSHHGAGIVFFNLPMILLLFMLFETAVRPALYDYL
PGRFPYRWRREGGIPATFKGWLTVAVLGLAGTGTHLIWDQFTHKGAWLVNHISFLGGTITAGSLEVPVYKAAQHGSTLIG
LIGTAWWIHRHLRVPAEERSRVSGKPFWFAVLALFVLIFAIAAVFVVDGKGLELLLQLVVPAISSLMLAVLLACLFFSKS
VSRNGRG
>lcl|NZ_CP026520.1_prot_WP_042227017.1_3809 [locus_tag=PC41400_RS19275] [protein=class I SAM-dependent methyltransferase] [protein_id=WP_042227017.1] [location=complement(4289929..4290690)] [gbkey=CDS]
MDKHFLDLVHGVKQDFTGWDFTSMTGTGRIDSGLLSWSYGSLAIPLVRSASTLLDMGTGGGEILSMLGPLPDQVYATESY
APNVPVARQRLEPLGVKVIQIELGQDSDLPFADKQFDLILNKHEIYCPAEMSRIITDDGIFLTQQSGGQDCRELNEYFGV
PLNEEFSGWNLDRAVEGLRANGFDIVMGKEEFIPQRFYDIGSLIYYLNATPWQVPGFEKNLYLEKLYEIHRIIGDKGYFE
TTQHRFIIKARLA
>lcl|NZ_CP026520.1_prot_WP_042227438.1_3810 [locus_tag=PC41400_RS19280] [protein=pentapeptide repeat-containing protein] [protein_id=WP_042227438.1] [location=4290964..4291848] [gbkey=CDS]
MSQPHENPETDSYKSPQGLLTDCEKCFGLCCVALPYAASADFAKDKDAGQPCPNLRQDFRCGIHTSLREQGFKGCTVFDC
FGAGQKVSQVTYAGEDWRRVPGSEKQMFEVFPVMRQLHELLWYLSEALTLQPARSVHGELRTALEETERLTFLAPEALLE
LDGAAHRGKVNELLLRTSELVRADAASRRKGGTQRRKVGRGADLMGANLKGADLSGANLRGAYLIAADLRNTDLRAADLI
GADFRDANLGGADLRGSLFLTQAQLNAARGDARTKIPAALVHPSHWSAAGHGTE
>lcl|NZ_CP026520.1_prot_WP_042227019.1_3811 [locus_tag=PC41400_RS19285] [protein=sigma-70 family RNA polymerase sigma factor] [protein_id=WP_042227019.1] [location=4292213..4292788] [gbkey=CDS]
MQSEYMYQLLTKMKDGDQQAFHAMYDATYQDVYRTVSFLVDHPQDREDVMNEIYMQMWTSLANYDPNRPFHFWLHGLVIR
QVQRFRVKSWRRFRIFERIRAFSREESHWDQPAVLMDGTNQMISQAIQKLTDKQRAVIVLRFYHDYTLEEIAALLDIPLG
TVKSRYHAGLQALRKNIWNLPPERMEKINDY
>lcl|NZ_CP026520.1_prot_WP_174446778.1_3812 [locus_tag=PC41400_RS19290] [protein=hypothetical protein] [protein_id=WP_174446778.1] [location=4292778..4293842] [gbkey=CDS]
MTIERKIREHLHEEAETMECPPAVTRRIEQSYSHYLQRKRSELPMKKRLIGGIVAAAILIPTAAYAAPTLIEMLTRTPMT
AEQVKVDKVGKAALEKLYNAFPETKSFEIIEASSVQGVQTSIILQEKGGTGKKITLHTNAVTGAIEHVIQENWEPKEKPL
PALTDQEIKSKTDYLIDKLYDSVKEYEFAMEQMENPDQKTFMLNYSKKGSKTPFYQVFVQGNAISVSLIGGETAPSTPST
DRVEGYFSRDGKPDYFADDFLNDAKLFSLLKISPAELKEELAKGKSVVEVASSKQVSKQQVIDVIAKTQVDSQIQSEKNG
DVPQGNSSIEQMLKAIEPKVLQVIEHKNETSSKK
>lcl|NZ_CP026520.1_prot_WP_042227023.1_3813 [locus_tag=PC41400_RS19295] [protein=class I SAM-dependent methyltransferase] [protein_id=WP_042227023.1] [location=complement(4294053..4294754)] [gbkey=CDS]
MNQDVLYDVWWKHHEEGDQGMEDDHIPHWKRVLSFIPEEDLSQCSVLDFGCNQGGFLRLLYEQRPFKEGIGTDLARRSVE
VANSRKGDLPIQYVATAEPEQFEGRFDLAFSLAVLYLIPDLGEHARKMKKALKPGGVYYATFADYSENPSLPYIQERINS
NASIPMQEHTLDHIAEMFFQEGFRVGIRRMIPTWYLDLSPGKRWYQRVADHLQYAYEQAYVLRFEAPGGPIGT
>lcl|NZ_CP026520.1_prot_WP_084706534.1_3814 [locus_tag=PC41400_RS19300] [protein=DMT family transporter] [protein_id=WP_084706534.1] [location=complement(4294787..4295650)] [gbkey=CDS]
MKNGILLSLLSSLVFSVMNALVKGISSTIPAAEIAFFRGLIGTLLILILMRYSGVAFSRKGIPMLAVRGLSGGLYMLAFF
YTLSNIPLTDASLLAQLAPIFVALLSALILKEKLGRRGAAVFPVIFAGALLLLKPHEFSSYSVYALVGLLSAFFSAAASI
SIRYLSRTHPSYEIVFYFVATSTLVAIPFMWDSFVIPDARESVMLLIIGLVSLLGQLFLTKAFTHENAVIVQIVSYFGLF
LNVLFGYLFWNEIPDLLTVAGGVMIVAGCMILFGGYRRSSPKTKLRM
>lcl|NZ_CP026520.1_prot_WP_084706519.1_3815 [locus_tag=PC41400_RS19305] [protein=ABC transporter ATP-binding protein] [protein_id=WP_084706519.1] [location=complement(4295670..4297394)] [gbkey=CDS]
MRTEQGERPPAAIGASLSEGELEQVTTSSDEPVPVLAIKELCICTKGSGRSLPVIQDISFSIRKGETLAIVGESGSGKSV
TAAAITGLLPSSLQIALGTINFQGDDLLTLRGRKKEGLLGRRIGWVSQDYHSSFTPLIKIGTQLTEMIRVHQKMPSHEAK
TLALSWLERVMLPAKRVFASYPFQLSGGQLQRAAIAAAFMLHPALLIADEPTTALDAVSADRVLGLLDELRVQTECAVLF
ISHDLRHVHRTADRVAVMREGRLLETGEAAEICRLPQHPYTQQLLDACPRIIPQGRLTGMGNIQILSESRTAPSPPLMSA
VGLVKSYPAKTGQEAAVRQLSFEIYGGECLGLVGESGCGKSTVAKLLLALERPDAGEVIFDGISLNGLTGRALRLVRKNI
QAVFQNPASSLNSRLPIWRSIVEPFDNFQKDDLSEFAGRGQSRRDIAAELLETVGLERTLGDRYPHELSGGQRQRVAIAR
AILLRPRLLICDEPTSSLDVTVQSQILNLLKQLRQELGMSCLFISHDIAVVQRMCTRILVMKEGRIIDRFPVDRLCSEER
HPYTRQLVSAFMRE
>lcl|NZ_CP026520.1_prot_WP_042227025.1_3816 [locus_tag=PC41400_RS19310] [protein=ABC transporter permease subunit] [protein_id=WP_042227025.1] [location=complement(4297394..4298287)] [gbkey=CDS]
MNGRIADNTAFAPNTAWNRLRHNRLACVGMLMIALFLLAALFGKWLAPHDPLLVSMSERLKPSSLQYPLGTDHLGRCIWS
RLLAGTSVTLGLSFLVVAVVMLIGIPIGILSGYRGGRTDAYFMRLADAAAALPEFLLAVTVTGFLGPGLMKVMLAISCIK
WISYARVVRGTIMSEKKQDYILASIVAGSSPWAVNRRHLIRHIASPLAVMAALDVGRTILLISALSYLGLGVQPPAPEWG
AMLSEGRPYFQAAPQFMIYPGVAILAVVIALNLLSDGLRDALDVQSDRPSRRKRRYS
>lcl|NZ_CP026520.1_prot_WP_042227027.1_3817 [locus_tag=PC41400_RS19315] [protein=ABC transporter permease] [protein_id=WP_042227027.1] [location=complement(4298284..4299228)] [gbkey=CDS]
MIKLAAGRIAYFACVLFFLSLVTFVLMKLAPGDPVRAVLKADEGAVSAVDVAAVKEKLGLNMPIYEQYARWFAGVLRLDL
GQSYTTGRDVVDILMERLPATFFLMAGAMLVVVLTAVPLGILGAKYQGRWPDHISRLIALLGISMPTFWLGLLLIYLFSY
HLNWLPVMGGGDLLHLVLPSLTLGFVMAPEYIRLLRSGLLDTLSQPYIRAARGRGIPEWRITVNHALRAALLPVISLFGL
SVGSLLAGSVVTESLFGWPGLGSLAMEAFTQRNYPIIQGYVLFSGLCIGLANLLSDISLGYMDPRIRYAKGREA
>lcl|NZ_CP026520.1_prot_WP_042227029.1_3818 [locus_tag=PC41400_RS19320] [protein=ABC transporter substrate-binding protein] [protein_id=WP_042227029.1] [location=complement(4299229..4300839)] [gbkey=CDS]
MYDSVKRKGGTQRRLSAISAIIVLCAVLMLTGCNPAAPLREGQPSETGDTAQRETGKKLTIVHSLPSDTLDPHNTWVPLR
AGVSETLVRLDEKMQVVPWLATKWETKDFRIWSFTLRDGIFFHDGTKLDAAAVKASFERGIADSSALSASLKVESIEANG
QELIFKTKEPFPAFPAELVHPTASVISTAAEKAMGKEAFNKSPVGTGPFKVASFIPGKEVALVRNAEYWDGKAKLSEVKY
QFNSDANVRTLALQSKQAGIVYHLAPEALAAIRQDGQLSIQSVTSLRTHYFTYNPKSATVQDIRVRQAIDKLIDRDAIVK
DVMLGHALPANGPFNSTLPFGRKGGYQKLDPQGALELLESAGYKKNAKGKLEKDGKPLTLKLIAFSGINPELPLIPQLLQ
SEAAKAGIELKIDSVEYPDVYIKDHNDWDLSTSSYLTSPRGDGGSFLNSAYVPGESYNPAGVEVEKLAPLLKTLNETGEV
EKRNMLTQQAVEVIMEAIPHSYIIYPNILVGINKGISGWKPGAEEFYIVTNKLDVN
>lcl|NZ_CP026520.1_prot_WP_042227031.1_3819 [locus_tag=PC41400_RS19325] [protein=opine metallophore biosynthesis dehydrogenase] [protein_id=WP_042227031.1] [location=complement(4300885..4302258)] [gbkey=CDS]
MTSPEQVVPPLGNTLIVGAGPAGIHIAVDVSRGWCGRLGLANRKGAHSTRILCELDRQGYTLSSVVEIQSKRHLTGETRL
DDYYDGFDGIDDRWHTVIVCTPSDSYSDVIGALRLENLSQVKRIVLLSPGIGSNLLVQSLVRPAGSDIEIISLSTYYGAS
RFAFEAGQSTVLRSIVRGLKRKVTAGSSKPDSRTLGDIKRFIESLGIQFESVSHPAAAESRSITTYVHPAFFINEFSMSE
IFSRTPSRKYMYKLYPEGPITPQAIKSMLLLWKEIGELVTRLHAEPINLLKFLNDDNYPVHETTLSREDIEGFTQADPVK
QEYLLYIRYASILIDPFAEPDEQGRYPDFSAFPYKQISKDSEGKWIIPRVPYEDYKKIKLVERLGKKLNLSMPQTSAFIA
EFESKLSSFVEEEGMEHFRPDLFTDTTESEADLILSGLELSWAAHATRHADLAIKVE
>lcl|NZ_CP026520.1_prot_WP_042227032.1_3820 [locus_tag=PC41400_RS19330] [protein=nicotianamine synthase family protein] [protein_id=WP_042227032.1] [location=complement(4302251..4303069)] [gbkey=CDS]
MKSKYAFLLSLQSLEYEINQLNAFSKEYPECFELLKTKLDDLCRFMVNDENEKRWHEWGTHDEVRQYSEKLRETSVKALC
DMEKYQSRLLINKELEATSYIEQLSHAVQEELDLFGIGPASRVLFIGSGAYPLSALTIAKESRAKVLCLDIDEEAVKLGF
QVASASGLQALVDFAGGRLSDHPFIREATHVFVASLVANKLEIVEEVKRTARQDTKIIVRYGNGLKSLFNYPLHADLSYE
WNLTPVVRTNSLYDTVLMESKLLTQERTIAFD
>lcl|NZ_CP026520.1_prot_WP_042227035.1_3821 [locus_tag=PC41400_RS19335] [protein=NUDIX domain-containing protein] [protein_id=WP_042227035.1] [location=complement(4303538..4303942)] [gbkey=CDS]
MSAAGIDKIAWIYVENGRILSARSKGKDTYYIPGGKRDPGETDTDTLVREVEEEVSVRIKPETVSHFGTFEAQAHGKEEG
VLVRMACYQADFEGDLSPASEIEELVWLAYGDREHVSPVSRLIFDRLHELKLLL
>lcl|NZ_CP026520.1_prot_WP_042227036.1_3822 [locus_tag=PC41400_RS19340] [protein=S-layer homology domain-containing protein] [protein_id=WP_042227036.1] [location=complement(4304059..4308618)] [gbkey=CDS]
MKTYQGTKAKLSFMLTFVLVFSIVLSAGQPAVYAGPNPPGLGADVQTGNALLESPDTPVPEDTYGGSAERPTPGVSPGPL
LLTAPAGGTENFALKKSVKVSGNEVDYGLYPEMAVDGDMGTRWSSAKTDDQWFEVDLGGKKEIGQVVIHWQTPAKSYRVL
TSVYGDVWENIRENDGLIQCEGGKEVLNFPSREARYVKFQGVERAPASDGILYGYSFYEFEVYKEDPIAKIIGGIQKLDP
VVKGQTSLNLPVMPPGYKISVYGSDALPVIDKEGTVHPPLTDRTVQLLLQVESESDPSRKGITGNIGVHVPGLYTQTPDR
NPEPKVIPSLREWAGGTGRLALSASSRIVISDPALRQTAETVRADLKEMTGLNPKIVQGKASSGDLFMKLDPAMKHLGAG
GYLLDVKESVTITAPETKGVFWATRSVLQILKGDPERRHLPVGTARDYPKYETRGLMIDVARKFYTIDFLRDYVKLLAWY
KMTDLQIHLNDDVGVPFKNGQRAAFRLESERYPGLASTDGHYTKEEFRELQRLGSDYRINVIPEIDTPGHSRIFTTYDPS
LGTDHTLDISKPATVDFVKSLFDEYIDGYNNGEPTFLGPDVHIGTDEYGGPSKEVFRGYMDMLIKHINSKGKHPYLWGGL
NEYAGATPISSEATMDIWYEPYGGPKQAMDLGYDIVNSDTNYMYLVPRLYRDYLNTPFLYRQWEPVKWPGVTLPYGLPRL
KGGMFAIWNDISYETGVSMDDSHVRMLPAMQVLSEKMWQGAREGSDYDAFMATAAGIGEAPGVNRLHRVNTDRPDGLVIR
YPFDGESEDTSGNKLHGKGENIAFTEGKYGKAAQFRGGKSYVETPIDTMGFGWTVSMWIKPERDNPDDAVLMESPMGKIK
LKQGKTGKLGFSKEEYDSVFNYEVPADKWTHLYLTGDAMGVSLYVNGNEYAEKLGLPVRRIQTLMLPVQKIGSETNAFRG
ALDNVHIYNEFLPLLDVNNLALHQTAESSPLEAPEHTPGNSIDGNLFTRWSSAYDDASWMIVDLGKSQTINKIQIGWETA
AEKFKLLVSEDKQTWTNVKGGDTILSSAGKLDTIDFQPVKARYVKFQGVERKPVDGTKYGYSIYEFEVYGDDKMVPYQEL
IAESAKLLSTGKGRPDIRRQVENLLLEYPVRFETVIDTLRDLNIRLKESIDHSGSDPDSGNPEDGDSGTGTGGGSSGTGG
AQPDGGNGLLTLKPGEGGTIRLGSEIGIEIPAGVSKETVTFGIRKITEPSGLIGGGLERVSPVFEVTRSAGTSLVKPIKL
TLAYEAGKAGRGRQAAVFTYDLVRKEWTRIGGAAAGGKISVEAPDTGIFAVFAEKEDGANRPPLFRDTAGHWAEEAVSRA
AAEGIADGYPDGTFRPDARVTRAEFTVLLARALHLEGTAALPGFADAADIGAWAGTAAAQAVEAGIVDGYEDGTFRPGAN
ITRAEMAVMAARALKLPAKPTAQTGFADDGDIPAWASGYVGAAAGLGLVAGPGGGMFMPGDPATRAEAVTLLLRAAGRK
>lcl|NZ_CP026520.1_prot_WP_042227037.1_3823 [locus_tag=PC41400_RS19345] [protein=hypothetical protein] [protein_id=WP_042227037.1] [location=complement(4309033..4309380)] [gbkey=CDS]
MELTSWFAGITKKRPGNGKSGRPGSPFLKESWAGYMSCSLAVVQLAVINLTLLFNVWTLDVLEYLFAASTLSALTGLFLR
NRSKKLCLWALAIQLVYVLFLAVVFVTGWVINPKP
>lcl|NZ_CP026520.1_prot_WP_241688698.1_3824 [locus_tag=PC41400_RS19350] [protein=anti-sigma factor C-terminal domain-containing protein] [protein_id=WP_241688698.1] [location=complement(4309371..4309802)] [gbkey=CDS]
MSLNRVTGTDETLKALQSGSVEVVWYAVDTGFEQRGVDSGGRVIGPIGYPAHEDWDAWSPFNDTKENAEQFMDTLVFLKN
YEKMAALFSRSQSLELDKRIRYLKDNGIKTYGAVVTGPVKDLAALRNNALVRGIKVGEVRLWN
>lcl|NZ_CP026520.1_prot_WP_241688699.1_3825 [locus_tag=PC41400_RS19355] [protein=sigma-70 family RNA polymerase sigma factor] [protein_id=WP_241688699.1] [location=complement(4309793..4310110)] [gbkey=CDS]
MNESSSLEDIYRKYFKDLYAYLFSLCKDHHLTEDILQETFYRAYVNLEDYSGTRIKPWLFTVAKNTYIDYLRKHKRIDLK
QDEFFAQVPVKGSAEQKRWLKPTCR
>lcl|NZ_CP026520.1_prot_WP_009674270.1_3826 [locus_tag=PC41400_RS19360] [protein=carbohydrate ABC transporter permease] [protein_id=WP_009674270.1] [location=complement(4310287..4311159)] [gbkey=CDS]
MYHKTLPYRIFSLFNNVFLTVLAILCLLPLYHLLMVSLSASAPANAGLVTFWPIGFTFEAYAKTFDNANFLSSLWVSVQR
TVIGTALALVVNTIAAYALSKESHVFRARNLYLWYFVVTMLFSGGLIPSYILILKLGLMNTLMALILPGLVGVFNIILLM
NFFRTVPKDLEEAAFIDGAGQFRTFLSVYLPVSLPVVATVSLFMMVGHWNSYFDGIIYIRDSEKLPLATFMQTIIVQADM
SKLDPAAVASLSQRTIRASQIFISALPILLVYPFLQRYFVTGIVVGAVKE
>lcl|NZ_CP026520.1_prot_WP_042227040.1_3827 [locus_tag=PC41400_RS19365] [protein=ABC transporter permease subunit] [protein_id=WP_042227040.1] [location=complement(4311175..4312125)] [gbkey=CDS]
MGVNTVQQVNAGKARRTDAKKGYKQPWILHFMVLPAEVLVFIFSYIPMSGIVMAFQDYKPALGISGSPWVGLKHFRYMWE
NDYFLKITWNTLFFACSKMVMNLIIPFFFALLLNEVRKMALKRTIQTLVYLPHFLSWVTLSGILIDMLAQTGLINQFLAA
VFGIKPIFFLGDGNWFRFTIIFSDVWKEFGFNTIIFLAALSGINPSLYEAAEVDGAGRWKQTFYITIPALIPITIVIATL
ALGNVLNANFDQVFNLYSPLIYQQGDIIDTFVYREGLLSGQFSFATAVSLFKSFISLILIAISYRLAYRFAGYRIF
>lcl|NZ_CP026520.1_prot_WP_042227041.1_3828 [locus_tag=PC41400_RS19370] [protein=histidine kinase] [protein_id=WP_042227041.1] [location=4312270..4314108] [gbkey=CDS]
MGLGLAGIVTKVRGKARYRFSLFAKMNTLIIVLFIPIILVYTYSNDVTFNVLSTELQKSNTKQLTFLSGQIDSRISQMMD
FSLVFSKDPNVSKFNGLNIWEDRYDRMQTRYVIQEKMALQSRVTDIWPARYAVYSQQNKDVISNYKSPAFDEQYLENNRS
GKWTYGDGSRKAPGGLDAFYWFYTDSLAQPGMLTGSSLVIEASFSYENIQNMLDTYKAGGQGDPFFYHKGDTPILNRSAD
KRLSEQLISYLDVHSPDNTTQHVVKLDGKNYLVSSVKSSYLDWHLVDVVPLDQILGPISFSRDLFYLSMFLLFVVGISAS
ILLYRNVQKPILKLINGLRRVQRGDYSVRLHTRDHNEFSFLFYRFNDMSRQIQDLIENVFNEKIRAREATLKQLQAQINP
HFLYNCLGYIINMAQMKEEEAVVAMAYNLSAYYRYTTRMERETASLEEEIKLLVNYLDIQKLRNGRIDYRIDIPEDMRSQ
SVPRLILQPIVENSVIHGVGKSYTSGEIRITGEMSDGFCRIYVDDDGPGLNAEQESALNRKMREPLQEEMGCGLWNTNQR
IMHQFGSRSYLAFVKSPLGGLRTEIVWEIPPAATTSDYSRPQGDLRHANDHR
>lcl|NZ_CP026520.1_prot_WP_042227042.1_3829 [locus_tag=PC41400_RS19375] [protein=response regulator] [protein_id=WP_042227042.1] [location=4314089..4315708] [gbkey=CDS]
MQMIIVDDEAHWVDNLALHKPWHELGIEQVHRAYSAFEALQLIDTHPIDIVISDVQMPEMTGIELIERIRIRDKKMKCII
LSGHSEFEYAKKALLHNAVDYLLKPPTDQELFGAVKTAMDQLNAEWELVSSLERTRHTLRENLPILRGQLLLDALQGRRL
SIDEWRRKLSDYGLPFHFENSALMLVRLEEEFGRYKNNGQPLIEYAIMNMAEEILGEFMEVWGVKEEHGYLVFLLQLKES
AGGAGQETLLEKLSQQLQHKVKQFLKGSLSIVITEWFAFPEQLPDRFRQASAYFREIVGDERDFVVRVNEVDHTPAQGPL
NALYVPPSLINLLEAGHWDAAEAKLTAVCTELEEKWPESWEHCMEAGFLIAASFSNLAHRSGYTLTKLLETGDMEALQSG
EAFVTIKKLGKWSLSVLHKLKEASSSSVKDSRSQYVKKIQDYADKHLHEDVSLRALADHINLHPTHLSKIYKIETGEGIS
DYISRLRMERACHLLKATGKKVYEISMEIGYMDPAYFIKVFKRHFGVTPQEYRDGALLP
>lcl|NZ_CP026520.1_prot_WP_042227043.1_3830 [locus_tag=PC41400_RS19380] [protein=ABC transporter substrate-binding protein] [protein_id=WP_042227043.1] [location=4315808..4317469] [gbkey=CDS]
MVKFRKWGSVLIAAGLVTLTACGGASGDTSKGSTSSSGAEAAFSKGKYDPPIEISSVLMPKKYVQGDTKENNVHDRWMLE
TLGIKHKDTWYPANDDQYKQKLQLAIASGEKLPDFVSVPTNPVLTNQLIDSGQFMPIDELFDKYANKILKDHSAAHPELW
YPFTRDGKKYNMPILEYTDNDDTLLWLREDWMEKLNLQAPKTIADLENIMDKFKNQNPDGLAPKDIFPLAISLKNNTNTW
MGSLDWLFGAYGTIQEQWNKDAAGSLEYGSTNPGAKEALAKLKEWMDKGYIHTDSALWDEAKSAESWTKGAAGILPGANW
VPDWPAPDLLKNVKGAKYKAYPIPAGPNGLIGTKWQNSGVNASIMINKDAKHPEAIFLYYNYLLDNLANPAAGSKYEYGF
AKGYDWDLIDGKPTSDKEKIKNFSNEFPFLTGPARIPDLYMKTLVKLADGKAPETPYEKQMAEFRKPENWAAAKIVMSQK
DIRKQNYFTGAATPTMVSKWNLLRQSELETFNKIIYGKLPITAFDEFVANWKANGGEQITKEVNDWYKSVSSK
>lcl|NZ_CP026520.1_prot_WP_042227044.1_3831 [locus_tag=PC41400_RS19385] [protein=SRPBCC family protein] [protein_id=WP_042227044.1] [location=complement(4317644..4318144)] [gbkey=CDS]
MLAILEKSPDGYVVRFDRSFGHSAEALWSMLTENDKLSRWFSELEVGELREGGFIRFNMPDGSFEKLDILDLHPPSILEY
TWGEDRVRFELYPEPEGCRLVLIEKLQAITAHTPKDLAGWHVCLEVVRLLLDGEEMPSRKEEWELWYEKYTRHVDAFLAE
EGGQGN
>lcl|NZ_CP026520.1_prot_WP_042227046.1_3832 [locus_tag=PC41400_RS19390] [protein=hypothetical protein] [protein_id=WP_042227046.1] [location=4318471..4319130] [gbkey=CDS]
MKKIGISLLSLFTFLFFASSAFAVTGEGAARLITTTKHTAIQGTLTIPSAWNVSNDGSYIAFYLGLGVVCEGGISFKPST
GWNKFLNCGGGEGSASNKTAPLTVQPKAGDTISLKLVNNLNDTATLYVNGVAAYTNLPVANAGALKAATTVKMVHSTLDN
QDKNSYTNASFSNVLVQSQSGGSYTTFPSNISPDFPLGGKGDYVINSSNPLGTTLKAGK
>lcl|NZ_CP026520.1_prot_WP_042227048.1_3833 [locus_tag=PC41400_RS19395] [protein=GNAT family N-acetyltransferase] [protein_id=WP_042227048.1] [location=4319392..4319811] [gbkey=CDS]
MYVKHERPGPVEYAELRKLAGLAPLEEAAAAAGLNGSIFTVVVRDEKSRLMGMGRIVGDGACYFQVVDVLVHPSCRDQGT
GTEDTLMKEIVHYLEENTSEGAQVIVISDTPGIRLYQSFGFKLMYPDFYGMSRNSRLPV
>lcl|NZ_CP026520.1_prot_WP_042227050.1_3834 [locus_tag=PC41400_RS19400] [protein=NAD(P)/FAD-dependent oxidoreductase] [protein_id=WP_042227050.1] [location=complement(4320050..4321099)] [gbkey=CDS]
MNEKDLFDVTVIGGGPAGLYSAFYSGLRELKTKIIEFQPHLGGKIHVYPEKMIWDVGGHTPLTGAQLIEKLVEQGLTFRP
EVVLNEKVESIDRDDEGNFVLHTASGQTHYSKTVIVAVGGGILKPQRLQIEGAERYEVSNLNYTLKSLQRFKDKTVIISG
GGNSAVDWANELESVAKQVYLTHRKDALTGHEAQVKQLMEGSAVCFLNSSITRLIAASSHDIIEYVELTNHLTGELTHVP
VDEVIINHGYERDSTLLKSSGLEIELADDFYIAGNASSVSTVEGLYAAGDILKHDGKLHLIAGAFQDAANAVNQAKQYIQ
PGSDKTGMVSSHNEIFKKRNRELVKQMMG
>lcl|NZ_CP026520.1_prot_WP_042227051.1_3835 [locus_tag=PC41400_RS19405] [protein=ABC transporter ATP-binding protein] [protein_id=WP_042227051.1] [location=4321296..4322087] [gbkey=CDS]
MVRLYTDELNIGYGERLIVKNLSVSFPDQKITTIIGANGCGKSTLLKAATRIISHQSGSILLDGELISKENTKVLARKMA
ILPQSPESANGLTVGELVSYGRFPYQTGFGRLSKKDYEVIDWALEVTGTSEFKYRPVDALSGGQRQRVWIAMALAQETDI
IFLDEPTTYLDMAHQLEVLELLQKLNLEQGRTIVMVLHDLNQAARFADYIIALKDGEVVKAGNCEEVISHGVLKQVFQID
AEIGRDPRTGKPMCCTYNLIKGD
>lcl|NZ_CP026520.1_prot_WP_042227052.1_3836 [locus_tag=PC41400_RS19410] [protein=iron-hydroxamate ABC transporter substrate-binding protein] [protein_id=WP_042227052.1] [location=4322092..4323012] [gbkey=CDS]
MKKLLLPILLVLVLALSACGTKRQDIADKGGASDSKQPATITYQSESGPVEVPANPQRVIVLSSFTGNVMALNVNLVGVD
SWSKKNPRFDKLKDVQEVSDESLEKIIELKPDLIIGLSNIKNADKLKQIAPTVTFTYGKVDYLTQHLEIGKLLNKEKEAK
AWIDDFKKRAAKAGDDIKAKIGADKTISVIESFEKQLYVFGDNWGRGTEILYQEMKLKMPDKVKEMALKDGYYALSAEVL
PQYAGDYVIFSKNPDADHAFQQTDTYKNIPAVKNGHVFEANAKEFYFNDPLTLDYQLDFFIKNLGK
>lcl|NZ_CP026520.1_prot_WP_042227053.1_3837 [locus_tag=PC41400_RS19415] [protein=iron ABC transporter permease] [protein_id=WP_042227053.1] [location=4323086..4324096] [gbkey=CDS]
MNKQNPRILPFSFKLAAGIVVFAGLFMAAMLFGAADVTVKDVWLALTSSLANEQITLLREIRLPREVAAIFVGAALAVSG
AVMQGMTRNPLADPGLLGLTAGANAALAFTIAFVPAVGYFGIMTACFVGAVIGAGLVFGIGTLKKGGFSPLRIVLAGASV
SAFLYAVAEGIGLVFKISKDVSMWTAGGLVGTTWGQLQAIAPFIAAGILIALVLSKQLTILSLSEEVAVGLGQKTLRVKA
VLFVVVVMLAGASVALVGNMAFIGLLVPHIVRAVVGTDYRFILPMSAVCGASFMLFADTLARTIHAPYETPAAAIVAMLG
LPFFLLIERKGGRALS
>lcl|NZ_CP026520.1_prot_WP_042227054.1_3838 [locus_tag=PC41400_RS19420] [protein=iron ABC transporter permease] [protein_id=WP_042227054.1] [location=4324093..4325100] [gbkey=CDS]
MIHPSLIRKQRLLVAGVIVLIALTILAGMGLGYSSLSYDRLVPTLLGQGSFKENFVLFSIRLPRIIITLLAGMGLALSGA
VLQGITRNDLADPGIIGINSGAGVAIAVFFLFVPIDPGSFVYLLPVVAFTGALITACLIYLLSYSRKTGLQPVRLVLIGV
GFSLALSGVMIVIISSAERVKVDFIAKWLAGSIWGADWPFIGAILPWLILLIPFVLYKANRLNLLGMSESVAVGVGVAVE
KERAALLLAAVALAASVVSVTGGISFIGLMAPHIAKSLVGPRNQMFIPLAILLGGWMLLIADTIGRNIIEPDGIAAGIMV
ALIGAPYFIYLLLKK
>lcl|NZ_CP026520.1_prot_WP_042227055.1_3839 [locus_tag=PC41400_RS19425] [protein=class D sortase] [protein_id=WP_042227055.1] [location=complement(4325358..4325996)] [gbkey=CDS]
MTKRIFPLLCVILGIAVFLYPTLQDRYENMQQSRLLDQWQANLKIIDQSEPENETDASSAGQAPNTAEDPISLATGGKHV
EGVLVIDKIQLKLPVLTDATVENLKISLASISNTGKPGQIGNYAIAGHRNFAYGKNFNRLDEVAVGDTIEVDTASDKFIY
EVKEKRYVLPQQVEVLQGNGKDSEITLITCHPMKNPTHRIIVKGTLIESRAV
>lcl|NZ_CP026520.1_prot_WP_042227056.1_3840 [locus_tag=PC41400_RS19430] [protein=collagen binding domain-containing protein] [protein_id=WP_042227056.1] [location=complement(4326124..4329918)] [gbkey=CDS]
MLKQKISAALVALLVFVQVFNGFGQWSAANAASITGSILDSVTLDVYDKSGNVVTNDVYEQGSKVQVDYHWSLPDGHGYK
DGDTFKFTLPAEFLLFNNISGPLLIESDDVGSFTVNKDTRQVVMTFNSYIETHDDVRGILTLKTQFDKQTITGSTLQKIK
FPIKGGEQIISLQFKPNVASTIGKSGVPQGYNAKNIDWTVDVNKELKSVENAVMTDPLPAGLSVPVTAAVYALDVNLDGT
VVQGALVDPGKYTLDTSGGTVKVSFADSPITTAYRIQFTTPLTDNAPAAFNNTATFKGSNQNAVSAGASVTVQRGSALSK
SSVSYDAATQTIDWAIQYNYNERAISQTNAVLTDLFDDVHELVANSVKVYPVTLDNAGNESVGSEISAGDYTVTSASAAG
KNGFKLQFNQPVTGAYKITYQTKTADRVIADGTVNNKVTSGSGDSATGTRTVKQVVLVKTMDAANYQDKTASWTLTVNGD
KQPMSEVVITDAFPNKGMKIVPGTFKVKQGSTVLNSPADYTLDGTVTEDSGFKLTFAGPLSETVTISYKTEYNFDWITPP
GSTKNFVNNADITWKDSTSAAQTKSVSAVFTPRNEAKNNGFKNGSYNASAKQLSWTIGVNYNGKPLNSPSVEDLLESKQT
LVPGSLALYKMTIAANGNPSLGTIIDSSEYSYSVDNANKLVVTLLNPSSTPFYITFKTSLDGELIESTVNNTAELFDAAS
PVSEKLTAKVTIPKGGEYVNKTGSQNGDKINWTININRGQSTIADAKIVDTPTENQLLLPESFRLYTTNVAANGDVVKST
EAVKGTDYNLVILTDDDGKQTFELDFLHEITAPYVLDYQSLIVANDQDTVSNKVSFSGNNVSVVSKETTNSIVVGVSSGS
GTGSGIRGSLTVKKADAADSSHLLQGAVFELYRKSGNTRSLIDSGTTDSTGTLVFKKLLAGDYIIKEKAAPSGYVLDNTD
RPVTINSTAGFTLNVTNAKVSVPTPTPTPTPPVTPTPTPPVTPTPTPPVTPTPTPPVTPTPTPTPPVTPTPTPSVTPTPT
PPVTPTPTPPVTPTPTPSSSPSDNSSSETPTPTPTPSVTPTPTPSVTPTPTPSETPTPIPSVAPTPTPSESPTPTPSETP
TPTPSETPTPEPTSSPATGKVTTDKDTPIGGKVDVPEGQKPLIDKQPEHGKVTIDENGRWTYTPNPGYTGKDRFTVAAKD
SQGNETILALLDVTVQTKPVPGQGNPPAAVLPQTGETGRWSVQAAGIALIVLGILWRRKLAVKK
>lcl|NZ_CP026520.1_prot_WP_042227057.1_3841 [locus_tag=PC41400_RS19435] [protein=response regulator] [protein_id=WP_042227057.1] [location=complement(4330313..4331461)] [gbkey=CDS]
MLKAIILEDEPLAALQMNRMLEAIPGIEVAGSYSDPDEAIAKAEGLQPDVAFLDIHLPGKSGIQAAELLQEVCPELDIVF
VTAYDKYAVHAFDLNAIDYVLKPLHKERLLKTVQRLTDKAQFFRRAREDDSSRSMRIRLLQTIQFERRNGTVERLKWRTA
KAQELFAYLIHHRGQAVRKSDLQEMFWPDLEAKKASAHLYTTIYHIRQALKDTGTDIGIRSMNIKEGYILDAGDIRIDSE
EWERDLTRLEAVHTGNAAEHRKVLDLYPGDFLADCDYLWAENERQRLRILWIEHARKLGEFYAGLGQVNEAILIYNRMQE
KYPHFEGSYFPLMKLYASLRDRKAVEDQFNRLCALLKEMDVLPEVSVQMWMEKWREGAAIVL
>lcl|NZ_CP026520.1_prot_WP_042227059.1_3842 [locus_tag=PC41400_RS19440] [protein=ATP-binding protein] [protein_id=WP_042227059.1] [location=4331713..4334835] [gbkey=CDS]
MKTYKFRVFIWVFLCFLSLGGLRLIWYHMTSIPANPPAVKGTLNLQNWDFAENPLVTLNGEWEFYPNQLIPVESGSAAPV
GMQVLQVPGSWEQSFAAEAKDSYRFGSYRLRVSLPRDTKLTFSIRLPDIRSASAVYVNGQPLHQTGRPAETGDGFIPGYR
PASVTFTADSGQANLIIHAASPSYMKAGGITGSVRFGEKTALTRTVWIGIGSQLAIALVLLIHSAYAVLLYLIGAQRRSV
LTFAALGLFAAVAVLSDNGKPLVVWLNMDYETATRTILVAYTGIALCLIQLARLWFLKQTGKRIFRMLQLIYVLYTLFIL
FAPAYMVSRSAFFLTVSGLATDLVFLVLMLRTIKREKEDGIFLLLGIAAVVSNVFWGALRPLTDPEPDFYPLDLFAALFA
LAAYWFKHYFRTSARTAELAERLQKEDKRKDDFLANTSHELRNPLHGILNMAQTVLHTEQDKLTEDSRKNMELLLTVGQR
MSLLLNDLLDITLLKENRIRLQRKKLKIQSVAGGVADMLRYMIGGKPLELKLDIPETFPLVIADENRLTQILFNLVHNAI
KFTNEGTIAIEARVSDGHAEIRIADTGIGMDGQQLIRLFQPYEQADSGITAPGGGLGLGLGICKNLVELHGGVLEASSVP
GQGSVFSFTLPLAEPFRAAAYPGEADISAPSGNAVLSAAPLAAAEEELTAAGPTGQVAHTARQEEAERPEKLPNILVVDD
DPVNLAVLRNLLASEGYRVETAASGREALSRLDGTGWDLVISDVMMPLMSGYELSRIIRGMYAPAELPVLLLTARSGPQD
STAGFQAGANDYVTKPVEAYDLKARVRSLIGLKLSVEERLRMEAAWLQAQIKPHFFFNTINSISILSEIDPAKMRALLEE
FSAYLQNSFDFQNSEQLIPLQRELELVRSYLAIETVRHGHRLDIHWEIADIPGLRLPPLSIQPLVENALIHGVLEKKSGG
VIRIEVKSRENGTEVRVTDNGTGMDPATLEKLKSLHAGEERRGIGLANTDRRLRHMYGAGLTLESSPGLGTAVSFFIPDN
>lcl|NZ_CP026520.1_prot_WP_042227061.1_3843 [locus_tag=PC41400_RS19445] [protein=DMT family transporter] [protein_id=WP_042227061.1] [location=complement(4334932..4335417)] [gbkey=CDS]
MILGLMLAVLAGSLVSLQTVFNSKVNERTGSWATTTLVLGMGFLASLTISLLVEGKQTFALQNMHPWYWVSGMIGVGVVI
CLVQGMKLLGPTYAISIVLTAQLSFALLWDSLGWLGLETVPFTSNKLIGVLVIVAGIVLFKFGGRQGAEPSAAKAPQGLP
D
>lcl|NZ_CP026520.1_prot_WP_042227063.1_3844 [locus_tag=PC41400_RS19450] [protein=cyclic nucleotide-binding domain-containing protein] [protein_id=WP_042227063.1] [location=complement(4335488..4336186)] [gbkey=CDS]
MKEIQDRELLSRYLREHRLETVFNEALNPHLSLFSFEQGEMICSRGEPAHYLYVLVKGKIKIYTSSAEGKTLILCFKTPV
EIIGDIEYIRGSDIVNTVEAVSPVHMIGIHYRWLKKYGNDHAPLLRFLLEIITQKFYIDSNFSSFNLMYPVEVRLASYLL
SVFCDESDNLVRGQLSTTHLMDAANLIGTSYRHLNRVIQKFCSEGLLQRTRGFITIKDREGLSALAGHNIYE
>lcl|NZ_CP026520.1_prot_WP_042227065.1_3845 [locus_tag=PC41400_RS19455] [protein=DMT family transporter] [protein_id=WP_042227065.1] [location=complement(4336239..4336664)] [gbkey=CDS]
MKGILFAFLGGACITLQGVANARISHDIGTWQAATITQLTGFLIALVILLFVRDGSLRSFKGIKPLYLIGGALGAVIIFS
EVTAIQKIGVTFTIAALLIAQLSLTFLVDINGWFGVAKQKMKLPQFIGIGLMIAGVVILKL
>lcl|NZ_CP026520.1_prot_WP_042227066.1_3846 [gene=acsA] [locus_tag=PC41400_RS19460] [protein=acetate--CoA ligase] [protein_id=WP_042227066.1] [location=complement(4336952..4338682)] [gbkey=CDS]
MTMQTEEQEIIPVLTKGNNLTDYEEAYANFDWADVEKEFTWSRTGKVNIAYEALDRHVEEGRGDKTALIYTDAARNESYT
FAQMKTLSDKFGHALRSLGIGKGDRIFVFMPRSPELYTAYIGAVKIGAIVGPLFEAFMEAAVKDRLLDSEAVAIVTTPAL
KDRIPVSELPALKHVIVVGENSEPGDREISYDDLMAKASDQLEIEWVGLEDGMQLHYTSGSTGKPKGVLHVHNAMIQHLH
TAKTVLDLQPDDIYWCCADPGWVTGTVYGMWAPWLNGTTSLVRGGRFSANDSYQTLETYKVTVWYWAPTAFRLLMAAGDD
LLKQFDLSSVRHILSVGEPLNPEVVRWGMKVFGKRIHDTWWMTETGAQVICNYPAMPIKPGSMGKPIPGVYAAIVDDQGN
ELPPNRMGNLAVRAPWPSMMRLIWNNPAKYDEYYRLKPWYISGDSAYKDEEGYFWFQGRVDDVINTSGERVGPFEVESKL
VEHPAVAEAGVIGKPDPVRGEIIKAFIALRDGYEPSEELIKEIQQFVKKGLAAHAAPREIEFRDKLPKTRSGKIMRRVLK
AWELGLPTGDLSTLED
>lcl|NZ_CP026520.1_prot_WP_042227443.1_3847 [locus_tag=PC41400_RS19465] [protein=ABC transporter substrate-binding protein] [protein_id=WP_042227443.1] [location=4338952..4339881] [gbkey=CDS]
MFNLKKSGVPAIFILLLFALTVGCGNKESAQPRQQTVTDGMGHQVNLPAKPKKIIASYLEDPLLALGIKPVAQWSVKGGN
VQEYLRSQLEGIPAIPSNLPPETVLSFNPDLIILRPEAAAQESQYEQYAKIAPTFVLSKEDADDYKKTITKMAELLNKKE
EAKKALDAYDRKAKESKDSLQKALGDKKVAVLWLTKKTFYVVNGNVSSGAVLYGDLGLNKPNLLNSLPESKANWVPVSLE
KLAELDADYLFLVNYDKGQAGNLEEDLWRNIPAVKNGHVYEIASTSSWLYNGVIAAEKIVDDVMKALRP
>lcl|NZ_CP026520.1_prot_WP_053228643.1_3848 [locus_tag=PC41400_RS19470] [protein=GNAT family N-acetyltransferase] [protein_id=WP_053228643.1] [location=complement(4340133..4341020)] [gbkey=CDS]
MPDTGMVLPVNRLTHDQLAQVKELEKRCNEYDSITLKLNWDMLENRKAEKCSDWLYVENGKTVGFLALFAFKSSEAEISG
MVDPEYRRRGIFRKLLEAADEECRKRGIPEQLLIVQRGSESGRSFAAYAGGAYEFSEYWMELPSDYDRDVLSSVISAGQI
ELRPAEQADMEVAIRLDVAGYGVTESDARGFNEQKLAAAGESFWIAELEGTPIGKINIQLYDGGYISGFCVLPEYRGQGH
GRVILGKAITIAESKQLQGKLALEVAANNDRALGLYKSVGFVERSVNDYYRVTVG
>lcl|NZ_CP026520.1_prot_WP_042227067.1_3849 [gene=uvrA] [locus_tag=PC41400_RS19475] [protein=excinuclease ABC subunit UvrA] [protein_id=WP_042227067.1] [location=complement(4341342..4343855)] [gbkey=CDS]
MSSEIVIRGARENNLRNVTLRIPKYKLVVLTGPSGSGKSTLAMDTLQRECQRQYMESMGMTSDTIAKPKVDAIIGLSPSI
SVGQHVTNRNPRSTVGTITDIHTYLRVIYEKLGERPCPNCGKRFVPGPDGEAAGPGDGEDGEDMQPAVKCPHCGVGMERL
TRRYFSYNTPEGACEECGGLGHTVTLNMEAVFDEEKSLRNGCVTIWYDVIAKYNLPILKAAGEHYGFTFDADLPLKEYGA
VQRDLLYYGVESEAFSSHFPDVKLPKSVGGGKFEGVVTGIWRRYKEKGGESGESALFLEETCAVCHGARLKESVRQVTVG
GAAISDISSWALQDALGWLQRLMGELQPEAREMVGSLLHDLTVRTERIVAVGLGYLSLSRQTVSLSGGEAQRLRLASILG
SGLTGVLYILDEPTAGLHPRDTAGLIGVLKQLRDLGNTVLVIEHDVEMMQAADHVIDMGPGAGSFGGRVVGEGTLAELMD
TPDSVTGAFLRESSGMTADLSRQRRPGNGRFLNVHDAYERNLKHIDVSFPLGCLTALTGVSGSGKSTLLFDLVAAGGRDE
QQRRGCRSITGFGGIGGIVTVDQSPIGRMSRSNVATYTDVFTAMRSLYAGLPEAKRAGLTAKHFSFNTPGGRCEHCQGLS
VVPMDMFFLPGLEVRCPVCRGKRFKEEVLRVTCNGYSISDLLNMTVQESLPLLQGRIKMDGQIALLTEVGLGYLHWGQSV
RTLSGGEGQRIRLAKELGGKTKRHTLYLLDEPSTGLHPRDTRQLLVLLNKLVDAGNTVIVVEHNLDMIRECDWVVDIGPE
AGDKGGEVVAAGTPETVAAAAASHTGRFLKKALAAPY
>lcl|NZ_CP026520.1_prot_WP_042227068.1_3850 [locus_tag=PC41400_RS19480] [protein=VOC family protein] [protein_id=WP_042227068.1] [location=complement(4344323..4344706)] [gbkey=CDS]
MKIEHMALWTADLEQMKSFYETYFGAVAGDKYTNAKKQFHSYFLQFESGARLELMTKPACAGREEGQIYLGYAHIAFSTG
SEDKVVELTERLRRDGYTVESEPRVTGDGYFESVVLDPEGNSIEITV
>lcl|NZ_CP026520.1_prot_WP_241688700.1_3851 [locus_tag=PC41400_RS19485] [protein=NADPH--cytochrome reductase] [protein_id=WP_241688700.1] [location=complement(4344703..4345065)] [gbkey=CDS]
MIDSGSLPCSGTPKTYVQHLMKENAAELIALLEQGGRLYVCGDGSRMAPDVEEEIRQAYANIRWVSPVEARKWLDDLQTD
GRYAKDVWAGIYSVEAQPEFAVMGAAGRSPAILRKGRRFV
>lcl|NZ_CP026520.1_prot_WP_042227449.1_3852 [locus_tag=PC41400_RS19490] [protein=VOC family protein] [protein_id=WP_042227449.1] [location=4345137..4345496] [gbkey=CDS]
MKVKRIIANLDTPELTAAKRFYQEALGLDLLMDHGWIATYGNRETMNVQISVASEGGSGTPTPALSIEVDDLDEALARME
KAGFPAEYGPVDEPWGVRRFFVRDPLGKLVNILSHGQED
>lcl|NZ_CP026520.1_prot_WP_042227069.1_3853 [locus_tag=PC41400_RS19495] [protein=DUF1772 domain-containing protein] [protein_id=WP_042227069.1] [location=complement(4345614..4346099)] [gbkey=CDS]
MSGSLLTVLTFCSALGSGLNAGLFFAFSAFVMTALARIPAEQGIAAMQSINATILNPLFGIVFMGTALLSGVLAVSSLLN
WGEPGSGWLLAGSLLFAAGFAVTMVFNVPLNNALEAVNPGSAAGADVWSSYVVKWTAWNHVRTAACLAALAMFIQALRLK
G
>lcl|NZ_CP026520.1_prot_WP_042227070.1_3854 [locus_tag=PC41400_RS19500] [protein=arylamine N-acetyltransferase] [protein_id=WP_042227070.1] [location=complement(4346214..4347113)] [gbkey=CDS]
MYTLTRDEVRAYLKRIGISEIGAPTLLYLAELHKAHVKHLSWQTLDIFGGKPAPIGFKESVALILNRRSGYCFHLNGAFG
VLLHSLGYRVNLHRAGVQPMGAEPRINSFHLGLTVDFSDEEAGRGTWIAEVGLGDMPFEPLPLRHGTFIQEPYTYKVTES
GVAADGWRLEHDHLASFPGVDYAPDVLTDLEEFKPNHEFYIRSEDSPWFNKFLIRQRNETEGNELRGCIWRKRNASGLEK
TELHTKSQWFEVLAGVFDEQLVNYSKPEREELWKRVEAAHSDWQKIKERQEKAGSLPTG
>lcl|NZ_CP026520.1_prot_PC41400_RS30395_3855 [locus_tag=PC41400_RS30395] [protein=cupin domain-containing protein] [pseudo=true] [partial=3'] [location=complement(<4347919..4348461)] [gbkey=CDS]
MNLAISYMDYTSPDLRFFYDLPKNTVFKTNSRNYINLLGYEQLNTLGNTSLLDIYLSKGYYVEPHYHQNATELVYCVSGA
AEVSFINPFTNKLYHIPIKTGQVANVPQGWWHYEQASEDDTHLIAIFDAPTPEVILGSDILRLTPARVMAETYCLDEEKW
KEAIAPITKTVTVGPPDNCRR
>lcl|NZ_CP026520.1_prot_WP_042227072.1_3856 [locus_tag=PC41400_RS30620] [protein=heparinase II/III family protein] [protein_id=WP_042227072.1] [location=complement(4348723..4351095)] [gbkey=CDS]
MTRERERERDRARIYEKRESRQEPADTSGPKRQSNPVPRCHGKTEPLLREEPGEVTAQSGAGRPAAASEPLSLAQGSLRD
KTARFPWAKETAEAIREELLRLNAAGSPAIPQEPGGWWHQYVCPVHHTELLFDPAEAEASSFRCPYGCVLEGAEYRGAWL
VFKHQSLARTALEAAAVHASLGDESCARLGKELLIAYAEQFPLYPVHAGAQGWMLKGRAFHQALTEAIWSTTLLGAYLLL
RDEGVSFSEEETAALGIFMGLLEESMTEYRHILIHERQNAENNYTAWLNASLSCVYVAQGRKDALEALIEGEGGFKHHLS
IGINADGFEFEGSTYYHVFVLRAYLISARMAERFGIDLYACEGGNGRSMRGMFEAMADLADDRGVLPALHDGPMERGPYA
REIAEIAEQGLSRYGIGGLRPILREAYRQLGGTLASPHARCGLEALLYGEDVQPDAAGEGGKPRRGARLWADTGFAAGRR
AGNPLSFFADFGGHGGAHGHLDKLHLSVMHPAGTLTPDFGVVPYGSPLRQAWYSETHSHNTVTLDGLSQAPHTGQCRRFE
ETEEALYAWFQSTGAYPGCTLDRHLLLTDDWLLDWFRVTTDDSGPRSIQWWMHSAAEPLPVSEEPGELPGKEHPAAAGTC
EGNVGANPVLLAESLKDVQFAGASGGKSSLAGKRDLDPAAPATVQIRYELGGESVYHTALVMPGDELITVRTPGDSVDPC
RLLTAIVHSHTGASADFIHVYRAGQAAELRRHQEGVVEVSVRKADSGETCARIVRLSGKTGLELMEPGGE
>lcl|NZ_CP026520.1_prot_WP_042227074.1_3857 [locus_tag=PC41400_RS19515] [protein=3-oxoacyl-ACP reductase family protein] [protein_id=WP_042227074.1] [location=complement(4351092..4351859)] [gbkey=CDS]
MAIDLTGKIALVTGSNAGIGRAVAIALAAHGAKVGINCLSNTAQGEEVVETIRAAGGEAVLVQADVTDIAQIDRLLSEVE
AAFGGTVDILVNNAGHLVQRVPNAEMTEDMYTRIMDVNFKSTVFMCKRVLPGMAAKGSGRIVNMTSVAAHNGGGPGSSIY
AASKGAVMAYAKGLAKETAGSGITVNNVSPGFIGNTMFHATFTTDTARQATINGIPLQREGTPEDVAGAVLYLASDLASY
LTGETIEVNGGMMMR
>lcl|NZ_CP026520.1_prot_WP_042227076.1_3858 [locus_tag=PC41400_RS19520] [protein=SGNH/GDSL hydrolase family protein] [protein_id=WP_042227076.1] [location=complement(4351861..4352916)] [gbkey=CDS]
MLHNGVYFHNVSELEQRPGLPGLLLQRFPREVREALSAKGRTKAAESSGCEIRFVTEAPYVRVTLGAREKDGRVYVFRGD
FFHAAYDLKAGTAVTLQLEEPERFAEVHRSALQSGAFSSDVWRVYCDRFTAVFYDLDAFGYEVHPPHKAEMPQLTLLAYG
SSITQGAGSLSHTGSYVRQAAVRLGVDAVNLGLSGTCYCEKEAADHLARRSGWDAAFLELGVNMRSVMDPPEFERRITYL
LDEVTGRNPHKPVFVTTIYPNRAAFFRDEEHVFTKRDLIFNDMLRRYVSDRNHSLLHLLEGSEIMTDFVSLTTDLIHPSD
YGHIRMGEHLARLIEPVITELRTSIRETGGI
>lcl|NZ_CP026520.1_prot_WP_042227078.1_3859 [locus_tag=PC41400_RS19525] [protein=chondroitinase-B domain-containing protein] [protein_id=WP_042227078.1] [location=complement(4353027..4356746)] [gbkey=CDS]
MIKSVPDFRARVRVFLSLQLVLSLCLSLVFVTSVSATETAGSEDLTVPAPAAGLTGPWLESFESYGSSQTGYTSPVGASP
TAAQPWTTVTGGSAGWSVGEEPASGSAEANHFLQQTDPTSGTSYLIANGYWGTDNPVLTDGSITLSGRVKVTGANSTYAG
LAAKYSVSGTKPTYYRFMTKKNSVSYQFYLERVTGTSKTAVPQTPGSPNASGVSVPNTVLNPFFDSQGYLPLKLDVIHRE
DGSLKLDGYYGGTLVLSGTDVSPLTAGGVGLYSQSGTTAFDDIQAAAYTGGGPGQPGVPSAPAGVSASPSGAGAVKLTWT
AVPSAAGYNVNMSANAGGPFTTLNASPVTATEYTAAGLTPGQTYYFTVSAVNASGEGGASAPVSAVPLASAKPPVTTTAE
LLTRLNAAVPGEVIELADGSYTGFTVRSRNGTAQNPIVVKAQHKGKAVFASTGLTVETSSYITIQDMEFRMNPQDKWINL
NGSNNIQITNNYFHSQNTTTAADKSNWIYIKGTNSHHNRIDHNLMENKRDRGKFILFDGVSSRPEGSTIPYEITRYDTVE
YNVFRNTLERQSNESEPIRIGVSVLSGLDAHATIQYNVFDHCDSDTEIVSVKSGANTIRYNYFVESLGSVTLRTGSGSSV
YGNMFIGKGRQVPSTDPDDAPLGTGGVRVYGENHKVYNNYFEGLTGTNWDATLAFTTGDNDNMTGPIDPTNNHYIAKNIV
IANNTLVNNKSGIELGMVRYGTAPKNLTFANNIVVGSQDELIRIMTPVPGLTWSGNIMFPQKGVPLVTGNSAPLTEDEVK
VVYPSLQNATLELGPQDYPWLWASSEYEPLRHIVFKKLAADSPAIDASKGNYASGGPLSFVTEDMDREARSGIPDVGADE
YTPDGRTDAEAPAWPSPNPLATDSVTPRTVKLKWTAAADDTGITGYRIYRSGALIDTVFGDVLSYEAGSLQPGTAYTFKI
EALDRAGHKTAGNTVSVTTPAFTGISLTGVPAQLALGGVPKQLSVVAHYSDGSAEPVPSGAVFTSSQPGVLAVSAEGRLT
PAGLGMSSVSAAYGGKTAPAVVYTVRSSSTNPFAIGGDTYVDNILSDKAGTNFSTEPEMQIKRKSASRRNGYIKAALPAL
AQPVDSVQLKLFVTSAQSGSDLQLTGILNDDWNPAEVTANNQPARSYSDIPLGSKTPLADGSYVIFDVTEFYNTPQAALD
GVLSFRLSMNENDREARIASLEGTDPAKAPVLLITTMNP
>lcl|NZ_CP026520.1_prot_WP_042227080.1_3860 [locus_tag=PC41400_RS19530] [protein=sensor histidine kinase] [protein_id=WP_042227080.1] [location=complement(4356913..4358721)] [gbkey=CDS]
MKKTLSRTIFIYIAVVVIITASSIGWFSYHISSSQLRQQNEQHLIEAVSNAVNHTDLYLKLYERAHLSLFTHPDIRKFLD
SPPKDDYAYFQYTDTLKKQVIDPLFLKTPEIMTVYLTDYSKNWVYFVNPALKIIQDPVPGATVEELDKETDSNGNLTIIN
NSVFTESDTQLITLSRKIHGQVFNEFKGILTIQIKSDEFSTLWKGVNLGPGGFFSIVDHEGRIVHHPDKTRIGKVIGGEL
LAKLENTQLTSFTDASFGEERMIVSRKSSYSGWNLMVSIPTEDINRPIASIRYLSFLVGGITLLIALLISYRFGRIITKP
IHILKRGMRQTEQGNWMKIPQLRTRNEFSELIDRYNLMVASLSELMDTLYKTELSKKDAELERQKAELQSLQLQINPHFL
YNTLENIICYAVIRQSEEITEIVDAIAAMFRYSVQTHIEETAIVNELKHVLNYMTIMKHRVGRDFELDVRITPEFFLKKT
ARLTLQPLIENVFQHAFADGIEDYHYIRLDAWTDEEERDLVIVIEDNGVGIPPEKLASLKRQLESNQLADSALGGRKGHG
GIGLMNVHRRIQLVFGEKYGLSISSIQGQGTSISIRFPNIGS
>lcl|NZ_CP026520.1_prot_WP_042227452.1_3861 [locus_tag=PC41400_RS19535] [protein=response regulator] [protein_id=WP_042227452.1] [location=complement(4358722..4359732)] [gbkey=CDS]
MKRILVVDDEPLIRQGLVAMLERLPYDFEWVRTAGNGQEALERMKESLPQLLLTDIRMPKMDGLELCRQVYDQYPLVQKV
IISGYGDFEYAQNCISYGVKEYILKPFTPAKIGDLLNRLISQGNPALSISKYEDLIGNIAESVWSMRIEEMETYFEQLKS
YCLTASRTDYDFVQLLKDCLPMLKKRLALKGFHPAVGEIPDSGDHLRYFEQLHVSLLKIIDELALQRGGQDLMLKAKDYM
DAHITKELSLEDTADYLGITPQYLSFLFKKVHKETFVQYRILKRIELAKKLLEIPRYRTNDIAYEVGYENYPHFSRMFKK
VTGVSPLEYRQQLGIK
>lcl|NZ_CP026520.1_prot_WP_053228644.1_3862 [locus_tag=PC41400_RS19540] [protein=extracellular solute-binding protein] [protein_id=WP_053228644.1] [location=complement(4359820..4361379)] [gbkey=CDS]
MKRWKKSALLLTLTASLALTACSGQGASPKESPGPVKVSLSLSDSLNKYALASTDINNDKWAKKLGEMTGVQLDIKLVPH
AEFKPKTALMFASNDIPDVMNTIQGMFPEGPDMIGAIKGGLFMPLDDLLKEYGQDLLKAIPKEAWDEVKYKGKIYAIPEY
LSVPTRRATFIRKDLLDATGLPAPKTIDEFLNVLRAMKKNGVEQPFAFRQNFVYSDFIFGAYDVMPYSTMFEKIGDEVVP
KFFKVEQMQKALQVYKTMYDEGLMTKDFASVDGNKWTNNINAGKSGIWNHNANLLMTWINTTKVTNPKSEVVIIPSPVGD
DGKGGMMKYSYTGGLSYINSKVSKEKAAAIVKMLNFMVTEQGDKFFNFGIEGDTYQTQDGKVVYKQPKTPEETYEEQFRS
VILRLVEDTALNRTLLNQTEDGKKLIQQFDTIVAKEGRDGISFAPELASSAKFTDAGLKFSDMPPVILDHMLKMVYGKEP
ISDWPKVLEEWKAKGGNDIIKEATERYNKKEGVKLGDSN
>lcl|NZ_CP026520.1_prot_WP_042227082.1_3863 [locus_tag=PC41400_RS19545] [protein=carbohydrate ABC transporter permease] [protein_id=WP_042227082.1] [location=complement(4361441..4362307)] [gbkey=CDS]
MSPLEKQSKTALLNPVLLLLISLAMILPILNIVAKSFSGAGALSSGSVLLWPSDWTLSNYATVLRQTAIWRAFGISALVT
VAGTLIALVMTASLAYPLSRTEYRERKGILVLVLITMIFHAPLIPNYLLIRSLGMLDSLWALMLPGAISAFNLFVMRSFF
QSIPNELIESARIDGAGELRILWTMILPLSKPAMATMSIIYSVALWNNYSNALYYINERALFPLQVKLREYVVTDSSDLA
NTAGDIANLSPEGLKMAVIVIATIPIMLVYPFLQKHFIKGMLIGSIKS
>lcl|NZ_CP026520.1_prot_WP_042227084.1_3864 [locus_tag=PC41400_RS19550] [protein=ABC transporter permease subunit] [protein_id=WP_042227084.1] [location=complement(4362323..4363285)] [gbkey=CDS]
MSYSGAGLGAEAKRQRTGKKAGFFSYYSRFKVLYLLSLPGIAYFLIFKYIPLTGSVMAFQDYNIFAGIWQSDWIGLDNFR
RMIEYADFWRILGNTLAIGFYTLLFGFPIPIILALMMNELRNAAYKRFVQTSVYLPHFLSWVVVGGIVIELLSPTSGAIN
YILGLFGREPVYFMGEDSYIRSIIVSSGIWRDAGWSTIIYLAAMSGINPDLYEAAQMDGASRWKQTFSITIPSIMPTITI
LFLLQIGNFLDLGFERVFVFLNPLTNGSGDILDTYIYRVGLIQREYSYTTAIGLFKSVIGFVLIVSANKFSKKTTGEGLY
>lcl|NZ_CP026520.1_prot_WP_042227085.1_3865 [locus_tag=PC41400_RS19555] [protein=DNA-binding protein] [protein_id=WP_042227085.1] [location=4363491..4363760] [gbkey=CDS]
MPTNLPDRPNKKAVASGPESDFPAGLSKPALRALAGAGFLRLEQLARVTEKELLQLHGMGPKGIETLRPALAAKGLSFAS
EPGASGTSS
>lcl|NZ_CP026520.1_prot_WP_042227086.1_3866 [locus_tag=PC41400_RS19560] [protein=TioE family transcriptional regulator] [protein_id=WP_042227086.1] [location=4363818..4364531] [gbkey=CDS]
MEYYKPIDIARKLHISTSALRHYESWGVVPAPDRAANGYRQYTRMHLAYFRCLRTLFPAFGVGLSCEVLRHIQRSDMDGA
FWLVSHEQSVLQHEKAVADKTLALLQNPELPILANKKLKSTMTIGEAASLAGVQDSAIRHWEKEGLLKPGRDPDNGYRIF
TPMHIRQILLIRTLRRTVYFLDRMKDIVQAVEHQSIEKAKKVTEQALHSIHERNRQQFRAVSELVELCKEVGLMERD
>lcl|NZ_CP026520.1_prot_WP_042227087.1_3867 [locus_tag=PC41400_RS19565] [protein=DUF4855 domain-containing protein] [protein_id=WP_042227087.1] [location=complement(4364773..4367301)] [gbkey=CDS]
MSSKKKWLSLALGAALMATMLPQAYGAQAQPEVRNLAKGLKYEWSEAPHSNYPDPSGKLTDGVIGSANVTDPSWVGSLNK
QTREIVFDLGEAKSISKIRAHFLQDWPGSAVLFPLTVSMYVSDDKQNWGTVAHKATEHLWSDGPPVDQFYTWDGSKDGIP
NAGSGAKMAYARYVKVAFSMHPKAWEFIDEIEIWGADGKIEGAKQVPAEAFDYMQPGENTAGIRNLSLLYNGYYSDGVKM
TKEKLVPEIGYVNKEGKPVDWFFDGALVLGLQSPQGRDFGEGEALLTDWKWYLDKTFAAQGDLDQLNEATKEVGQKLGQP
GHKTKTVMMIPNPGEYTTDFGDIDGDGVSENFNEGAVGKDKAIANRQKVIRWWIDEVSRKFEEKHYSNLELSGYYWLQEQ
VGTSSTGPDLLRFTNGLVHEKGLKSFWIPHFLAYKSFMWKDVGFDAVTFQPNYYFEPLNIDRLQDAAETAKRFGMGVELE
FDDRLLADAVFRKRFLEYLDAGEQFGFKGSSFKAYYKGSGPVMAQTAASSDPEIRGLYDKLYEFTKGGAPVGGAPVASDS
VISTAPDTPVSGTLQVTDDDGDALTYSIVRNGAKGNAALTDASNGTFTYTPGKGQTGTDTFTFKASDGKTDSNVATVTVT
IAAGQTGWQTVLTGSAAVPAGEKFSVTYGLTGGTQAVYAQDLKVEYDAAFMDFVSAKPVLDGISLLETDKKTPGQLRLLV
ASQGDSHAVTGPAKLLELTFRSKKDNQTIKGVVSVTSAVMGDREGKEAEAAPSSLTVQTPGSVPGDYNGDGKVSVGDLAI
VAAPYGKTKQSPDWEQVKHLDINKDGKINNQDLSAVASKILK
>lcl|NZ_CP026520.1_prot_WP_042227088.1_3868 [locus_tag=PC41400_RS19570] [protein=YncE family protein] [protein_id=WP_042227088.1] [location=4367760..4368830] [gbkey=CDS]
MSKAILIQRKVAPKVIATIPLNSPLHVAINPNTNRVFVTRSLDALAVINGLTNRVIKTIPIENAPNRLVVNARKNRIHTL
NTGVTANPTGITGSVSLVNGATNRLVSSVDVGKDGAAIALNAANGRVFAVNAGSNSVTVINGRTNRVITTVKVGDTPFNA
AVNPITNRIFVSNHDASTVSVINGATNKVIATVKVGSGPDTIAINPLTNRIFVANTNSSTVSVINGKTNKVIKTIRVGVS
PNDIKVNVRTGRVYTANSTANTVSVISERKLAVIDTIKVGSQPSVIGIDTKKNRIFVVNASLKLSRGTVSVIGGCSNKVI
ATLKVGAQPSDLSVNSKTSRVYVTNSAGDSVSVIRS
>lcl|NZ_CP026520.1_prot_WP_042227089.1_3869 [locus_tag=PC41400_RS19575] [protein=lytic polysaccharide monooxygenase] [protein_id=WP_042227089.1] [location=complement(4369006..4370055)] [gbkey=CDS]
MTIRLPYQGMIKMLAAGAIMLMLSIAMLLLLENKASAHGYVSNPSSRAALCASGVNKNCGLIIYEPYSLEALKGFPAAGP
ADGKIASANGLFAPLDEQSSTRWTKVNLSPGPTTFNWTLKVPHATAAWKYYITKQDWNPNAPLSRASFDLTPFCNVPYKG
QPSGSYSDTCNVPSRTGYQVILAVWEISDTANAFYNVIDVNFGGSPGTPDTTAPTAPAGLTASNVAATSATVSWTASSDN
VGVAGYRIYNGSTQIGTTSGALSYNLTGLTANTAYAITVKAVDAAGNVSAASNTVNFTTIAGTTYPAWNASTAYTGGSKV
TYNGVNYEAKWWTQGETPGSNSSVWKVIP
>lcl|NZ_CP026520.1_prot_WP_042227091.1_3870 [locus_tag=PC41400_RS19580] [protein=basic secretory protein-like protein] [protein_id=WP_042227091.1] [location=complement(4370609..4372222)] [gbkey=CDS]
MKKLTYLTATAFLGLSLIGSFGVPTAEAAQTTSTTTPKNVAAGGSVTASGEYAPTQTKESAFDQSLYSKWLTFSSTGWIQ
VKLDTAKTVTSYTITAAEDFPERDPKSWVLKGSNDGSTWTVLDTRQNEDFTYRNQTKSYSLANASSFKYFKLDSITNHSG
GILQISEIKLFDGSVDTWTTIKPTVTASAENTPNEGKDNLVDGTSVTKWLTFNATGWLKFDFGKPVTIDGYALSAANDFS
ERDPKSWVLQGSNDNATWTNLDSKSDESFGLRHQRNHYVLNGNNTAYQYYRLNNITNHSGGILQLGEVEFSRKDDMWHAV
NPVIEVQNLDAAGNGQLFDQALPNATDDIRVIVRKVNELLYNNPSESQGVQKIKVTIEDVPGVAWASGDHIQKTVGFSSR
FLRDIANEPGKSVRDEILGVLYHELTHCYQYDDNRYADKSVGGVSYMVEGLADTIRFTVGYHNRYNMTKGGTWQDGYGTT
GNFFRWIEDTKKPGFVRELNASLTPFDGQDWTEGKIQELTGTNVNTLWTEYQAALPN
>lcl|NZ_CP026520.1_prot_WP_042227092.1_3871 [locus_tag=PC41400_RS19585] [protein=LLM class flavin-dependent oxidoreductase] [protein_id=WP_042227092.1] [location=complement(4372448..4373806)] [gbkey=CDS]
MTKRTDQLHLGAFIYYAGHHHAGWRHPESGVEGMYDFDLYRSIAETAERGKFDMIFFADLLYVQHVDMAASGMLDPITLL
SALAAVTKRIGLTATVSTTYNHPYNVARKFASLDYISKGRAGWNIVTSQLDIEAHNYGLPKHPDHALRYEMAEEFVEAVT
RLWDSWDDDALVLDREAGIYADPGKVKAVDYQGKWFSSKGPLNVPRPPQGNPVLIQAGSSRQGQEFASRYGEVIFTAQQS
LEAARAFYAAVNGRLSGAGRKPGSLKVMPGISPILGSTVEEARRKERELTDLIPTELAVMALSGMVSHDLSAYPSDGPLP
DIPDPVEKSNGMKSRVQLIMDTARSEGLSIRELGQRILGARGHLQFVGTPEQLADLMEQWFQEKACDGFNVMGPVLPGDL
ETFTDEVVPILQRRGLFRTEYSGTTLRDHLGLERAEAGHFSRRGAAEVGVNG
>lcl|NZ_CP026520.1_prot_WP_053228645.1_3872 [locus_tag=PC41400_RS19590] [protein=ABC transporter permease] [protein_id=WP_053228645.1] [location=complement(4373841..4374707)] [gbkey=CDS]
MGKSVSVSAKGSGQTFSGKTAPNVYGRTVKDLGWLGVFPFWTHGLLVPFVLLAVWQAAGMSGLVGKEQLPTPSDIARRFA
ELGLNGELLRHIGVSFVRAASGFGLGALAGLLTGLFTGIGKMIERTLDPSLQMLKTVPLLALIPLFILWFGIGEFSKILM
ISLAAFFPVYLNTFTGIRGVDSKLYEVTRILRYSRIQQLRKLIIPSALPHILLGLRLAMGVSWLVLVFAELMGTSAGIGY
MIEDARTYSNTDIVFVGIILFACVGSLSDLAVRALERRWLGWRDTFKG
>lcl|NZ_CP026520.1_prot_WP_084706520.1_3873 [locus_tag=PC41400_RS19595] [protein=ABC transporter permease] [protein_id=WP_084706520.1] [location=complement(4374710..4375567)] [gbkey=CDS]
MLSASTRNKLMDGKRFGGTPGAGKSRKKSLLAVNRLAGLLLPLSLLAGWQIAGSAGWLNPILLPTPADILGEFGAMLLSG
ELFRNLGVSAWRVLSGFLLGGSLGLAAGLWVGFSYKAERLLNPMFQMLRTLPHLAIAPLFILWFGFGEESKLLLIAKGSF
FPLYIQTFLGIRSVDNKWFDVARVLEFSRFQMIAKLIMPASLPNILLGTRLSLGVAWLGLVVAEMMGSSTGIGFMINDAR
SMSLTTVMFVGIVVFAVMGKLSDSLVLLAERRLLRWRDSFRGEGG
>lcl|NZ_CP026520.1_prot_WP_042227093.1_3874 [locus_tag=PC41400_RS19600] [protein=aliphatic sulfonate ABC transporter substrate-binding protein] [protein_id=WP_042227093.1] [location=complement(4375591..4376610)] [gbkey=CDS]
MKIWTTALLLFVTVLIAAGCAKNGETSPSANGGKPDYKGLTLKLGVQGSGGMYAKAREEKWFEKAYEPLGVKVEWAEFQS
GPPMTEAMASDKLDFASLGNMPVIAAQSAGIKFEMISQILDGKRNTAIIVPANSPLKDIRDLKGRKVAVTKGSNAYNLLT
RGLHDAGVNPADVQVIQLQPNEAQPAFDSGKVDAWATWDPYITVNTLTGKGKVLAEGETLGVLAPSFVIVRKEIADKYPE
LATVYLSVLEKTRLWEEQNSAEVLKRYAADYKIPEAVVGGMLTRSKSINIPVSEATAAELQKTADFQESAGTIRKKIKVA
DAVNNTFIEEALKAASAVK
>lcl|NZ_CP026520.1_prot_WP_042227094.1_3875 [locus_tag=PC41400_RS19605] [protein=TetR/AcrR family transcriptional regulator] [protein_id=WP_042227094.1] [location=complement(4376909..4377505)] [gbkey=CDS]
MSPRNAERDLRVREERVQLILDAAITVISSKGFGAASISDIAKAAGISIGTMYHYFSSKDEIFADVLKRGQLDYGKRAAE
LADEQTMPPLVKLHEFAAGWLANPINWAYTIMLQSARTTETVPAEIRQAVTDRFTANLQPIARIMKDGQGRGDIIAGDPL
QLAFYYVSLIQGLTLQRAPGFEIPVSIDADALIRFFEA
>lcl|NZ_CP026520.1_prot_WP_042227095.1_3876 [locus_tag=PC41400_RS19610] [protein=transporter substrate-binding domain-containing protein] [protein_id=WP_042227095.1] [location=complement(4377675..4378562)] [gbkey=CDS]
MRKTIAIWAAVLLLAGVLAACGAVKDSKDATRSGAKPASAGESGGGSIGKIKDRGKLIVGVFTDKPPFGFTDDKGNKIGF
DSDLAKRFAKDLLGDESKIEFVTVEPASRIPFLQSDKVDLIVANMTVTDERKEAVDFTNPTMKVATQVLVSEKSGLKTLA
DLKGKKIIVTKGTTADLFLTKNRPDVELVKYERNTESLQALKDGRGDGYAQDNFVLFAWSRQNPGFTLIPEKLEKEAPIA
PAVKKGNTELRDWVNKELETLGQEKFLLKLYNTYVKEELGASTDPNDVITEGGKL
>lcl|NZ_CP026520.1_prot_WP_042227096.1_3877 [locus_tag=PC41400_RS19615] [protein=amino acid ABC transporter ATP-binding protein] [protein_id=WP_042227096.1] [location=complement(4378559..4379326)] [gbkey=CDS]
MGERLLQLRQVEKSFDGHRVLKRIDLDVAAGEVVVILGPSGCGKSTLLRCLNGLEPIQGGTLLFKGEDLNAGKVPWRVTR
QKIGMVFQSYHLFPHMTVLDNILLGPLKVQKRSRREAAADAEKLLSRVGLLDKKNAYPRQLSGGQQQRIAIVRALCMRPE
VMLFDEVTAALDPEMVKEVLEVMLDLAREGMTMLIVTHEMRFARAVADRIVFMDGGEICEIAPPDAFFNRPQSGRARQFL
NRFFEIETGKGREEV
>lcl|NZ_CP026520.1_prot_WP_042227097.1_3878 [locus_tag=PC41400_RS19620] [protein=amino acid ABC transporter permease] [protein_id=WP_042227097.1] [location=complement(4379356..4380024)] [gbkey=CDS]
MALSAYELLFRSLPQLGEGLWATLRISLLTIVFSVAGGIGFGLLRTSRLLLPRLLSRVILELVRSVPILVWLFFFFFGLP
ILFDIHLPAFASALLVLSLWGMTEIGEIVRGALQSLPKGQTEAAKSLGLSGRQLLVYVLVPQAVRRMAPPAINVFTRIVM
TSSLTVLVGVPELIKSGQQIIERHYTLSYASVVIYGLMALLYFALCYPLSVFSRRLEKRWAG
>lcl|NZ_CP026520.1_prot_WP_084706521.1_3879 [locus_tag=PC41400_RS19625] [protein=amino acid ABC transporter permease] [protein_id=WP_084706521.1] [location=complement(4380005..4380670)] [gbkey=CDS]
MHLDTDFMWKHLPDFVRAGGETLRIGGIAIFFSLFVAVIHSFILFFRIRPLTWAVRLHIELARNTPLLIQLFFLYFALPQ
LGIKLSAMTTVFIAMSFLGGGYFTEVFRSGMEAVEKSQSESGLAIGLSKWQLFRYVILPQALRISIPSLFANFIFLLKET
TVVSAIAVPELLYTTTSYISLYYKTYEMLLMMTLCYLVIFLPLSFLLGYLERRYRHGAFGL
>lcl|NZ_CP026520.1_prot_WP_042227098.1_3880 [locus_tag=PC41400_RS19630] [protein=GNAT family N-acetyltransferase] [protein_id=WP_042227098.1] [location=complement(4380932..4381333)] [gbkey=CDS]
MDIRYSHSNEVDADKLRQLFLSVDWESGKHPEELHQAILNSHSVVTAWHGDKLVGLVNALSDGVMTAYFHYMLVRPDYQN
QGIGKELMTEILGKYREYKTKVLISYAEAESFYRSLGFRAEKGAVPLYISELV
>lcl|NZ_CP026520.1_prot_WP_042227099.1_3881 [locus_tag=PC41400_RS19635] [protein=VOC family protein] [protein_id=WP_042227099.1] [location=complement(4381358..4381696)] [gbkey=CDS]
MKLNHLNLTVTDVPASQAFLETYFGLTTRSTRGSAFAVLSDDNGAVLTLMKGAEVSYPKTFHIGFIQESEEKVNEINRRL
REDGFDVEPPQRMHGWTFYVKAPGGFVVEVLS
>lcl|NZ_CP026520.1_prot_WP_241688701.1_3882 [locus_tag=PC41400_RS30410] [protein=SRPBCC domain-containing protein] [protein_id=WP_241688701.1] [location=complement(4381749..4381976)] [gbkey=CDS]
MVDPFRKLRMTWKRKEWPNPSRLQITFTAAKSGGTTISIHQEMLDDVYMRELMRREWESVLNRIQELGAEEYKRV
>lcl|NZ_CP026520.1_prot_WP_241688702.1_3883 [locus_tag=PC41400_RS30415] [protein=hypothetical protein] [protein_id=WP_241688702.1] [location=complement(4382013..4382369)] [gbkey=CDS]
MPGALFKENIRMETGITWEEWVVKLQRSVNELWSHEEIRDHICRCYGVSGEWGEWLAAMYAPYLGRTPVGVTKDAGVQIG
VRKTAAISVPLAWNYLTSREGLALWIGSVSAFPLEKGA
>lcl|NZ_CP026520.1_prot_WP_042227100.1_3884 [locus_tag=PC41400_RS19645] [protein=amidase family protein] [protein_id=WP_042227100.1] [location=complement(4382392..4384056)] [gbkey=CDS]
MMNIHGQRPDTLASSDISSPEAISHVSYAKLPELGAAEMARGIKAGRWTSRELVEAHIARIRQIQPLIGAVAVPLFEEAR
RAAAEADRLTASGEPLGPLHGVPVSVKESLDAAGTPSTWGLAGRTAPSSEDDPAVAALRRAGAVVLAKTNAMQLLMGCET
VNPVFGRTSSPWNPAGRSSGGSSGGEAALIAAGGSPLGLGTDVGGSIRTPAHFCGVHGLKPTPGRVASRPPAGIVHICRE
AAEMASTGPLARKVEDLALAMDVLAAPETLQVHPLQRPGETRPAELRIGYYTSDGILPPSPAVERAVLDAAEALRARGVT
VKPFRLPEPALALQQFYALMSAAGAEGIAETAGTDALVPQVEGIRRSFGRSRTANRMLALILRGVGQKIAGTHIVPCLGR
KTRAELGQAVELRQAYREKFAAEMERQQVDVLLSPPFLTPAIPHDHSLAMTYEGSYALLYNYLGMPAGVVSLSQVRDGEA
AVMRVVRSRDRLVRAAAQADPNSAGLPVGVQVAAAPWREDRVLAVMDMLEKLFRDRPDYPPGKLAAALLPIAGN
>lcl|NZ_CP026520.1_prot_WP_241688703.1_3885 [locus_tag=PC41400_RS19650] [protein=PadR family transcriptional regulator] [protein_id=WP_241688703.1] [location=complement(4384053..4384625)] [gbkey=CDS]
MNIEQVILGFIYGEAMTGYELKKRFDESVAHFFGATFSGIYPALRRMEKEGLIEKEVVPQEGRPSKNIVRITEKGRQSFA
DYLKSPLSPGVQKSDLLVRYFFGRYAAEGQIRGWLLEERERMTDTREGLLHMRRGALNDPETDAFRLKTLEMGIAQIEFS
LRWIDEELAMEPGAELTGPESREERRRCES
>lcl|NZ_CP026520.1_prot_WP_241688910.1_3886 [locus_tag=PC41400_RS19655] [protein=cupin domain-containing protein] [protein_id=WP_241688910.1] [location=complement(4384816..4385385)] [gbkey=CDS]
MPNLAFNVRDNVLFKRNEDNVAYEVTSTQLPPMTGGAFTDLRLGRGHIREPHWHPNAWELHVVVAGEADISVMCPDSPQL
VNYRAKEGQVVFVPAGWWHWITPVSEKVHLHAFFNHGQPETVHGSDILRLTPPEVLKNAYNVNAALAGKAFAPIMESVRI
GPPVSVPVQSEPGKPANPGQPYLNEKKKT
>lcl|NZ_CP026520.1_prot_WP_042227462.1_3887 [locus_tag=PC41400_RS19660] [protein=TVP38/TMEM64 family protein] [protein_id=WP_042227462.1] [location=complement(4385609..4386223)] [gbkey=CDS]
MDISDIVSYVTNEDNLKQILEQYRSLGPFPGILLTFMKSFVPPLPTLLIVGVNAAVYGLWAGFLYSWIGIVSGCLVTFLI
VRKIAGHPYLERLARKPKIARTTKWIRRNAYSYVFLLSLFPVGPFVVINIAAGLAGMRLRSFLMAVLIGKGIMVFTVSYI
GYDVAKYIENPYQLLYVLLFVVVSLIVSKKIEKRFSSGQTMELE
>lcl|NZ_CP026520.1_prot_WP_164977141.1_3888 [locus_tag=PC41400_RS29890] [protein=hypothetical protein] [protein_id=WP_164977141.1] [location=4386465..4386626] [gbkey=CDS]
MVHTKTKSYTYKVGKNTFLVIKIFQTAVAKTRNGNALASNAANVKVFKKDKKR
>lcl|NZ_CP026520.1_prot_WP_164977142.1_3889 [locus_tag=PC41400_RS29895] [protein=hypothetical protein] [protein_id=WP_164977142.1] [location=4386660..4386833] [gbkey=CDS]
MSKHRKNRKHVKHTKVKTTRNTVNVKQAAKARTGQGGNAFAINASDVGVVRTDVAAK
>lcl|NZ_CP026520.1_prot_WP_042227102.1_3890 [locus_tag=PC41400_RS19665] [protein=cytochrome P450] [protein_id=WP_042227102.1] [location=complement(4387009..4390215)] [gbkey=CDS]
MNKEGESVLAKSEPLIPQPKTYGPLGNLPLIDKESPLQSMMKLADELGPIFKLEMPGRTQLLISSHELVADACDESRFDK
KIGAALSNVRAFGGDGLFTAHTDEPNWQKAHHILLPTFSQRAMKGYHDMMVDIAVQLIQKWSRLNADEIIDVPEDMTRLT
LDTIGLCGFNYRFNSFYREQPHPFITSMVRALHEAMNQTQRLGLQDKLMVITKRQFNHDIQAMFALVDKLIAERRAQGTE
GIDDLLSHMLNGKDPETGESLDDENIRYQIITFLIAGHETTSGLLSFALYFLLKNPDKLRKAYEEVDRVLTGPVPTYAQV
RELNYIRMILNEALRLWPTAPAFVLYAKEDTLLADKYPLQKGDSLTVLIPKLHRDKSVWGDDVESFIPERFEDPGKIPYD
AYKPFGNGQRACIGQQFAMQEATLVLGMVLKHFNIRQHTDYVLKVKETLTMKPEGFTMKVEPRHQGAAVTIPGAAVSAAE
DKQVKEQAAPLSVNAHNTPLLVLYGSNLGTAEGIARELADAAKYQGFRPEIASMNDYAGKLPKEGAVFIVSASYNGKPPS
NAKDFMDWLESAEPSEIAGVRFAVFGCGDHNWASTYQDVPRRADELLASKGAVRLAGRGEGDASGDFEKQLEDWREALWP
DVMKALGLELAEISEKEKSSLSLQFVSGFAGTPLAESYDAVHALIAGNTELQHEGSGRSTRRIEVKLPEGVTYRVGDHLG
VLPRNSRESVARVLRRFGLQGSDHLILSGSGRSAAHLPLDRPVSLDDLLGHSVDLQEAATRAQLRELAGCTACPPHKREL
EALLEEDAYREQVWTKRVSMLDLLEKYEACELGFERFLELLPPLKPRYYSISSAPQAQPEEASITVGVVKGPAWSGRGEY
RGVASNYLAERREGDAIVMFVRTPESGFGLPDDPKTPVIMVGPGTGVAPFRGFLQARSAMRREGTELGEAHLYFGCRCES
DYIYREELEGYEREGIVRLHTAYSREEGTPKTYVQHLMKENAAELIALLEQGGRLYVCGDGSRMAPDVEEQLRQAYANIR
QVSVDEARKWLDDLQTDGRYAKDVWAGI
>lcl|NZ_CP026520.1_prot_WP_042227103.1_3891 [gene=dhaS] [locus_tag=PC41400_RS19670] [protein=dihydroxyacetone kinase transcriptional activator DhaS] [protein_id=WP_042227103.1] [location=4390548..4391138] [gbkey=CDS]
MTDSQTTKKALAASLKKQMETKPLAKISVQDISSGSGLNRQTFYYHFKDKYDLVHWIYYSEAVESIADYRDYGHWSDMAV
QILHYLMKNRLFYVNALNTPGQNAFDGYLFEATQDMIKRVVDDASGGLDVSDADKCFIAEFYTFAFVGIVVKWVKTGMKE
LPETMMKRISDLVDGTLTRALSRHTGMQTQPVEAAP
>lcl|NZ_CP026520.1_prot_WP_042227464.1_3892 [locus_tag=PC41400_RS19675] [protein=MFS transporter] [protein_id=WP_042227464.1] [location=complement(4391277..4392506)] [gbkey=CDS]
MFKNAFVRTIILSNVLLQLGVWVRNFAILLYVTDITNNNPKSVSLISVAEFAPIFLFAIIGGTFADRWKPKRTMIVCELL
SALSVFAVLLVVMYGSWQALFFATLVSAILSQFSQPSALKLFKQHVPEEQLQGVMAMFQTLMAIFMVFGPILGTYVYQHY
GLEISIAVMGVCFIASAAVLTSLPADKENPSKAESNFKQELKDGLRYVWNSRALKTLGGTFAVSGLAVGIIQPLALFIAI
ENLGQDKLFLQWLLAANGTAMLVGGGFVMALSKKMRPQSLLAAGLIASTVGTIGIGLSTSVPLTIGFEVLTGLFYPCIMI
GINTMMMKNTEAAFIGRVGGVLTPMLMGFMVIGMSAAGVMKAAFSLQAVYAGSGILFFLGALLLLPLMLRHPESPLESQG
AAAVADSSD
>lcl|NZ_CP026520.1_prot_PC41400_RS19680_3893 [locus_tag=PC41400_RS19680] [protein=YkvA family protein] [pseudo=true] [location=complement(4392881..4393280)] [gbkey=CDS]
MGQEKMIDRVKAKAREIKRNIFVLYLAYKHPRVPWYAKLIALCVAAYAFSPIDLIPDFIPISWVS**PDNRPVRYYTCSE
ANSARSNTGL*GKSRRDPEER*TQELGDWGIVHYDMDIACNLDRFYRL*IFQI
>lcl|NZ_CP026520.1_prot_WP_042227104.1_3894 [locus_tag=PC41400_RS19685] [protein=DUF3139 domain-containing protein] [protein_id=WP_042227104.1] [location=complement(4393497..4393853)] [gbkey=CDS]
MRNISRGKIIALSIVLIVLLAPFVYVQGNKIVYAERVKSYLIHEKSYKEEELASVKGVWGMKLPSFYTVVVFKDEPGVDY
VYFSHNDVLQFDQRITPKGVEMGTMPEKLKHTEKRKEG
>lcl|NZ_CP026520.1_prot_WP_042227105.1_3895 [gene=yqeK] [locus_tag=PC41400_RS19690] [protein=bis(5'-nucleosyl)-tetraphosphatase (symmetrical) YqeK] [protein_id=WP_042227105.1] [location=complement(4393841..4394476)] [gbkey=CDS]
MNVYEHLVRPELSGDLKKDVYAFLTENGHRETAEHCMKVGEQARELAVRYGADPETAARSGYLHDISAVFPNDARISVSV
QLGIEVLPEEESFPMIIHQKLSKAIARDLFGVEDPEILSAVECHTTLKKQASQMDKLLFTADKVAWDQIGTPPYLEELNR
GLALSIDHAAFAYIRYLWSRKDTLRVVHPWLKEAYADLSRHIESESQTCGT
>lcl|NZ_CP026520.1_prot_WP_042227106.1_3896 [locus_tag=PC41400_RS19695] [protein=aminopeptidase] [protein_id=WP_042227106.1] [location=complement(4394796..4396028)] [gbkey=CDS]
MANQFEKNLEKYAELAVKVGLNIQPGQSLWVNASIQQAPLTRFIAQKAYEAGAKSVHIEWMDDVITKLRYTLAPDDSFDT
YPAWRAQAVEELAETGGAYLLIDSGNPELLQGVKPERISAYSKAAGKALAKWREYMTSHKLTWSILAAPSTGWASKVFPD
LPEDEAVAALWEQIFKATRVDQDNPVQVWERHNATLRTKVEELNAKQYRKLHYRAPGTELSITLPERHIWSGGSTANEKG
DAYNPNVPTEEVFVSPLKDGTNGVVRSTKPLSYQGNLIENFTLTFENGRVVDFQAEKGYEALKGLLELDEGARHLGEVAL
VPHESPISASNIIFFNTLFDENASNHLALGKAFGFSIPEGTKMSVEELTKAGLNDSITHVDFMIGSAEMDIDGELADGTI
EPIFRKGNWA
>lcl|NZ_CP026520.1_prot_WP_042227107.1_3897 [locus_tag=PC41400_RS19700] [protein=CGNR zinc finger domain-containing protein] [protein_id=WP_042227107.1] [location=complement(4396152..4396703)] [gbkey=CDS]
MENLALEFMNSQWYDWRGTGRSEDRLDRPGWLEAFLEKWELQAQIPPQPRERTALGELRKAMRAITECASSGREPDHGDI
ARLNEAMAEVPSRPVLTREEGGWRLESTTGDRGWTAVMGRVAVAYARLLAEEDVRRLKICANEDCRWIFFDESRSRTARW
CDDKMCGNLMKVRRFRKRRKEND
>lcl|NZ_CP026520.1_prot_WP_042227108.1_3898 [locus_tag=PC41400_RS19705] [protein=DinB family protein] [protein_id=WP_042227108.1] [location=4396902..4397396] [gbkey=CDS]
MNGLDLLKRGWEHAYGTEDWYAPLKDAVSGISAEQASWRPPGEAGNTIWETVNHLVYYKQRLLEHLQGEEPDNWAQNNDV
TFKTPDTHGSEEAWQRTLGELDAVHTRIGSLLGTKSDSDMDAVYRDSPIGFRISSIIQHDAFHTGEIVLIRKLQGSWPAT
RHYE
>lcl|NZ_CP026520.1_prot_WP_042227109.1_3899 [locus_tag=PC41400_RS19710] [protein=cyclodeaminase/cyclohydrolase family protein] [protein_id=WP_042227109.1] [location=complement(4397540..4398160)] [gbkey=CDS]
MRVYDQSIREFLFEASSESPVPGGGSIAALAAALGASMGSMTAQLTTGAKFHDKEAAMLENADLFRQAIVEFESLQEKDM
ASFNGFMKALKLPASTPAEKEARSLAIREASAYAAAVPLELMEACLAVMRRLAELAPAVNKNVVSDLGIAVLMLEAAVQS
AWITVEINLNTMSGDSRNVYRDKGTLLLRESEEIKRDALAQVRERI
>lcl|NZ_CP026520.1_prot_WP_042227110.1_3900 [locus_tag=PC41400_RS19715] [protein=bifunctional 5,10-methylenetetrahydrofolate dehydrogenase/5,10-methenyltetrahydrofolate cyclohydrolase] [protein_id=WP_042227110.1] [location=complement(4398157..4399014)] [gbkey=CDS]
MGMSMKTADIAGRLRDSIRIETDAWKKRGIEPSLAVVLIKGDPASEYYARAKEKTAHKLGIRYRLTEFEPGITQSELLGH
LGELNEDKSVHGIMLELPLPRHLNPSEASGLIRPDKDVDGITPGSKLACFLGTRGLYPATPQSCIRILKEFGFELKGRHV
VLVGRGETVGRPLMPMLLREHATLTVCHSHTENLARHIAGADILITAAGRAGMITADMVHPDLVIVDAGINETDTGLTGD
VSPDALQRVKAMTPVPGGVGTLTTLILFENLLKAVGMQIGEGGLK
>lcl|NZ_CP026520.1_prot_WP_042227111.1_3901 [locus_tag=PC41400_RS19720] [protein=MFS transporter] [protein_id=WP_042227111.1] [location=complement(4399267..4400463)] [gbkey=CDS]
MSVAADIKLFISVKKAWVLVLAMILYGIGVGILAPMNAIYLKEGIGLGKGEIASIFALSLVLNMIFTISVGILSDKIPRK
KIIPMAAAVVCAAGLLIYMRADGYVSALVGMSIASAPSGMIMGQMFAMARNHFSKLAPDIVEISQLWLRASFSVGFFTGL
LLGANIYLWFTFYGVLWGNLGGYTALFVLLLFYRETVEAPSKAVVKKSGEPFSLAMLAALLMLSCADAIRGLYLPLVVNE
KFGNPELMSYIWSAQAVFELLFMTVAGYWALKYGFKPVILFAGLFALIVYLVYANVDYLPFFFAMQPLYSFYVAVLLGVA
MGYVQRMFLHKTGFGASLYVVISQLASLVGYFLPLLIEGITPSIFYIPAALIVVSMLLITRTMVKERGSRKPGNVQTL
>lcl|NZ_CP026520.1_prot_WP_042227112.1_3902 [locus_tag=PC41400_RS19725] [protein=lytic polysaccharide monooxygenase] [protein_id=WP_042227112.1] [location=4400937..4402001] [gbkey=CDS]
MSQSISWNRSLFKVSPFLVAFAALVIGAICSLVFAESASAHGYITGPASRAALCNQGINKNCGAIVYEPQSLEAPGSYPA
GGPADGQIAGAGAAFPELNAQTADRWSKVNINSGSNTFTWKLTAAHATREWKYYITKTGWNPNAALKRSDFELFCSFNDG
GKKPNPSTSHTCDVPARSGYNVILAVWEIADTPNAFYNVIDVNVNGGGTGTPTDNQAPTVPAGVTASGVTANSVTLNWSA
SSDNVGVTGYDVYQGSTLVGTSTTTSFTVTGLAADTSYTFTVKAKDAAGNTSAASSAVTAKTAAGGDGGTTAPAWSASSV
YTGGNRVTYNGVVYEAKWWTQGETPGKADVWKTV
>lcl|NZ_CP026520.1_prot_WP_164977143.1_3903 [locus_tag=PC41400_RS29900] [protein=hypothetical protein] [protein_id=WP_164977143.1] [location=4402186..4402323] [gbkey=CDS]
MGKKMMFTAIGLGAAYLLRNKDARQKLKDQFQSFTGSAKTQQTLQ
>lcl|NZ_CP026520.1_prot_WP_042227113.1_3904 [locus_tag=PC41400_RS19730] [protein=GTP-binding protein] [protein_id=WP_042227113.1] [location=4402603..4403682] [gbkey=CDS]
MTIPVFVLSGFLGSGKTTLLLRMLEESKKLGLRPGILMNELGKADVDGLILEGHSSASIEKLLDGCVCCSKKSELTGSLQ
ALLVRKPDILFIELTGVANPEEIADVLTEPGLLGRVALKQVITVLDAENFMDYNSLFASDKQLVRTLRRQVEVADLILVN
KTDYVHRIRLHQIEKAIRKLNGHAMLVYTTHCRVELKPILEGVQPVQKGPLTVRPIRMTKITTHPKEPSTAGTGRGMHPA
DSPEPAETAPSHSRVRTVTLPWNQAAGLTKADVDAFLQRWKDGLIRAKGYVTFAESSHMHLLQYAGKRSYWEPSFYQGEP
YLVFIGIDLDTDRLSEEWQRLQDTPQPQSLEPESGRRRK
>lcl|NZ_CP026520.1_prot_WP_042227114.1_3905 [locus_tag=PC41400_RS19735] [protein=TIGR03943 family protein] [protein_id=WP_042227114.1] [location=complement(4403861..4404802)] [gbkey=CDS]
MEDRRVYAFHSFMRAAILCGFSFYIVYLVKTGGMSHYLAPHLVPYVKASAIALFVLAVIQVYLALQPAAGNESDCGCEPG
VRKSKLSHGAAYSLFLLPLLLGFLLPDRVTGSDIVQVKGIQLNASNLQTQRNAGTRDDSPEKQQSSEEEASRGKEDTEEA
RLKEMFKGSRFTEDYAKLAMRLYQNKTILIGEKGFMERLTAVDLFAGNFAGKQMEISGFVYREDDMPKNQFVVARLTMQC
CSADASPYGVIVEAPGADSYLKDTWVTVKGTIRKSVYEGKDVMKLEGAVIQTIKEPSSPYVYPYAGDFADLGK
>lcl|NZ_CP026520.1_prot_WP_042227115.1_3906 [locus_tag=PC41400_RS19740] [protein=permease] [protein_id=WP_042227115.1] [location=complement(4404830..4405852)] [gbkey=CDS]
MHFSGASRGMRKRHLIYIAAGSFLAVLAWQAVRTGNAGLPDPSVLRLFKTMFIGIVLEAVPFIMLGVLLSSFMQVYIPET
WIRKAIPRNPLLGFVYACLLGILFPICECGMIPVVRRLINKGMPLYIGVIVILAGPIVNPVVYAATYTAFRMNPEMVYAR
MGLALGVSAFIGLFVYAFINQNPLKHRLDSLYAENGQDCAHVQGGGKFVTMMGHAGGEFFDMGKYLILGSLVTAVIQAFV
PRGDLSALGEGEVLPHLFMMGFAYILSLCSTSDAFVASSFLQTFSKGSLLTFLVFGPMLDFKSTLMLLSVFKTKFVLLLA
VLITMTVFAGSVIVEHLFFN
>lcl|NZ_CP026520.1_prot_WP_042227116.1_3907 [locus_tag=PC41400_RS19745] [protein=class I SAM-dependent methyltransferase] [protein_id=WP_042227116.1] [location=complement(4406071..4406961)] [gbkey=CDS]
MYYGVKLNTNYKTLWQEGMKDWDGNLPVRMTDDVKEEGFWNTFMKKKEQAGVDSWVANLRKELITLIDPGEDVLEIGPGW
GNYTFEAAEKAGSLTCVDSSESVLAFLKASAERKGWNHVNFVHAKWEELPEVKRADVVFGVNCYYRMQEIDRALLHINNT
AKRLGIAGMVSGPERPHYKDIQEQLGCRIKFQRRDYIHLQNLLYELGIDANCKIVNLHKTYAYDTEEELLKDNLSAVLDK
EYDRKKAVELLYAYVREEDGKLIYPHNFKAALLYWKPEKRLEPADLPTEVKQVVGV
>lcl|NZ_CP026520.1_prot_WP_042227117.1_3908 [locus_tag=PC41400_RS19750] [protein=phosphotransferase] [protein_id=WP_042227117.1] [location=complement(4407541..4408476)] [gbkey=CDS]
MPNNSSLISKTVLIAEKAGLKDITPLELGNGGNLIVHLAPYPIVARIAAVLSKNNADYAYKVLVRELRAARHLHSKGVPV
RLPTDRTDAGPYDIGGTWVTFWQYVPPTELQRPSPSEAVELVNRLSLALKDFSDEIPMLGVWERTCQSAFRLMKHPDQRI
QNLLSLLFAIDRQMRLETSLLIPSHGDAHSRNLIPSPEGWVWSDFEDLSLMPAYWDLASYVGNLALFNGIQEPTFQYIWS
GINNQADKQAFGFAVTARILMSTLGNLDFALSGSGDLEFAYKQLELAKDFIDQINLIIEGEQRSHNRHGST
>lcl|NZ_CP026520.1_prot_WP_042227118.1_3909 [locus_tag=PC41400_RS19755] [protein=spore coat protein] [protein_id=WP_042227118.1] [location=4408682..4409170] [gbkey=CDS]
MSLPKVPQHLAWHETLELHELVSLQTGQLIACKQKLPTLADEGLQTLFRAAIQTLEQNLNELLAYYPEAPAASRNKNAEL
PVLEAGQLLAFAKTAVRSYACALTETATPRLRATFVKHLLRAVEFHAKVFNRMYADGAYPAYDLKELLENDAANARAALS
LE
>lcl|NZ_CP026520.1_prot_WP_042227466.1_3910 [gene=bshB2] [locus_tag=PC41400_RS19760] [protein=bacillithiol biosynthesis deacetylase BshB2] [protein_id=WP_042227466.1] [location=complement(4409259..4409915)] [gbkey=CDS]
MERHILVVLPHPDDEAFGISGTLAGHIRQGTPVTYACLTLGEMGRNMGNPLAANRVTLPEIRKAELEESCRSIGIQDLRM
LGFHDKTIEFEDRELLVGKIRGLIDELNPSLVITFYPGYSVHPDHDATGAAVIDTVQRIPVEERPVVHCIAFSKGCEDVL
GKADVVNDVRPYLGQKLGSILAHSSQFQLNGQLGGKSANDKEVQERFGYERLWTYRFQ
>lcl|NZ_CP026520.1_prot_WP_042227119.1_3911 [locus_tag=PC41400_RS19765] [protein=YojF family protein] [protein_id=WP_042227119.1] [location=complement(4409942..4410286)] [gbkey=CDS]
MQFINQAAVQQVLDLLKGQDLYIHLEMTNGAYASHLDQSKHTASTFIRNSKIRYLQGSIAGNGPYRVGLKMEEGWVYSEG
LTHWDETETQRLILSGHDKDGKLVVSLQLSRDPF
>lcl|NZ_CP026520.1_prot_WP_042227120.1_3912 [locus_tag=PC41400_RS19770] [protein=PAS domain S-box protein] [protein_id=WP_042227120.1] [location=complement(4410476..4413940)] [gbkey=CDS]
MNIPSQAKETLFTHLYGQAANGLVFASPENGEWLGMNPALCNILGYSEAELLERTEKLLLHPDDESFISEHKRMMEKLAA
FPGEVSEREKRYFHRDGRVISLHVRTTLARDPEAGQPLYLIHEFTDMTDKNAAAKELEDNRDLALLFSKNIQDVISFSSP
SGTILYISPSVEGLLGYKPEEMIGRKRVDFYHPDDALQMRNRDKLYSEKEVFVRRVRHKQGHYIWLETAFQLVRSGDGRV
IKILAIGRNVTERKKYEETLAEAQRVAHIGSWEWDLLNSRVSFSDEMRRIFGYAIHSADSDAELFLSCIYPEDLPRVRES
IEKALENGESGEMIYKICLPDQSIRTIAGQWEVVADEFNKPIQIIGMVQDITERRRMEERILESEQNYRLISEYSLDFIS
RHAVDEQATFLFVSPICHTMLGYEPEEMVGTAGLGYIHPDDVDFVKKYLQENMHGEGDEKVTFRFFRKDGTYVWFETTSR
YTLDDQGREQEIVAISRDITERRNTELRLQEYKSLFDYNPVGVASLDLHGKLLTANRGQQVLTGYSEKELTDRHFASLVH
PDDYDKTLRHFELAAQGEPQTYETGLIHKDGHRIEASVINVPIVLSGHIVGVYVLTSDITDRKKYTEEIEKLSYEHALIL
NSVSEGIFGLDLQGQGMFINPAGASMLGFNPKELIGNRHLLTVEQACPDGNQYLPGDSPVERAVRDGVSVSPSEGIFWRK
DGTSFLASYQVTPLFDKGERKGVVVVFRDVTNEKEIIRAKEFAEQADRAKSEFLAIMSHELRTPMNGIIGMTGLLQETML
DEEQRSYADIIRESSDALLHILNEILDFSRIEAGKMELNLDAIDLRETLAGVLELFRVKAEEKNLKLSCDIPAGLPRYVL
GDEARVRQVLINLISNAIKFTERGSVSVAVEITPSPEPQSVMLEFKIRDTGIGIPAGKIHLLFQSFSQLHPAINRKYGGT
GLGLSISKKLVELMGGEIGVESEENAGSTFFFTLYASLGDSEAPDTASAIHPAETAPVPSLPEAKPAPLRVLIADDHPMN
RKLLHTLTGKLGYSADVVNSGAEAVQAAGEQTYDLIFMDVQMPVMDGFEATRLIRERHGKKAMPPVIIAVTAFAREEDKE
QCLASGMNDYISKPVLSGELKRLMAKWSARIKER
>lcl|NZ_CP026520.1_prot_WP_042227121.1_3913 [locus_tag=PC41400_RS19775] [protein=hypothetical protein] [protein_id=WP_042227121.1] [location=complement(4414208..4415095)] [gbkey=CDS]
MKSALYNGHDGSHSMFSGERVLIWTGLAGFVLAAFMAVYMALYGSRVLPEGNVESAFSFNAALGIFILSIAAVLPLASMG
SRSRKVYRWMLAFTGGAAYLLESVQHLRGINPRFTQAGTDIDRIAGIVFGLDSMLLIVLVVWLAAAFYRKSVLAERPLVV
LGIRYGFLSTMAAVVAGIGMIALQSRYTGASGNLIVLHGLGFHAVQTLPLLGWLLEKSDDAQSRKVRLVHAGSIGWMLAI
AGVGFQTASGMSVAELSPWTLTAAAGLLVWLLALGTALRKSVTVTWRVRPGVSKL
>lcl|NZ_CP026520.1_prot_WP_042227122.1_3914 [locus_tag=PC41400_RS19780] [protein=TetR/AcrR family transcriptional regulator] [protein_id=WP_042227122.1] [location=complement(4415191..4415709)] [gbkey=CDS]
MVVREDRKEQIINAAVGIFAELGYYKTTTAMVAKAAGVTQPYVFHFFKSKEELFKAVADRAVQRLYDAFMQVEAQGNLIM
ESMGQAFKDLMRANRAEMLMVLQAPAISEPEIRRHMSEKFALIHSTIMRKFEESGMDDVETAAGNFMAIGLLIVTSDVLQ
LPEMLPFKDSNC
>lcl|NZ_CP026520.1_prot_WP_042227123.1_3915 [locus_tag=PC41400_RS19785] [protein=manganese-dependent inorganic pyrophosphatase] [protein_id=WP_042227123.1] [location=complement(4415840..4416769)] [gbkey=CDS]
MEKTLIFGHKNPDTDTICSALAYADLKTQLGENVEAVRLGAVSGETQFALDTFKAEAPRLVETVAGEVKTVILVDHNERQ
QSVSDIDQVRVTEVIDHHRIANFETSGPLYYRAEPVGCTATILNKLYKENGKSVPANIAGLMVSAIISDTLLLKSPTCTE
QDVAAARELAEIAGVNLESYGLDMLKAGADLSDKTIEQLISLDAKEFQMGSHKVEIAQVNAVDTNDVLARQDELEAALNA
IIEKKGLDLFLFVVTDILNNDSIGLALGSSAKAVEEAYNVTLADNKAVLKGVVSRKSQIVPVLTDILSK
>lcl|NZ_CP026520.1_prot_WP_042227125.1_3916 [locus_tag=PC41400_RS19790] [protein=class I SAM-dependent methyltransferase] [protein_id=WP_042227125.1] [location=complement(4416896..4417708)] [gbkey=CDS]
MKPIVNYYSQFDEWGRLDREPWEYTINLSFIKKYLPAGSAVLDNGAGPGKYAMELAREGHDVTLADLTSKLVETARSKAA
ELGLLDRFRGFHAADARDLSLFGDGQFDASLMLGPLYHLQGAEDREKAVRELHRVTAKDGWVFVAFMPRIRHLANSLRDP
LNWKPNDTVRGLEAFAETGRFDHSDAGRFTGAYYFPVGDIVPFMETNGFGTVKLIGSGSIAAGMTGEQWDYWRSQGEEVS
RRVLDLIMQAAEDPYNLGSSSHLLYVGRRL
>lcl|NZ_CP026520.1_prot_WP_042227127.1_3917 [locus_tag=PC41400_RS19795] [protein=GNAT family N-acetyltransferase] [protein_id=WP_042227127.1] [location=4417861..4418316] [gbkey=CDS]
MNITELRHGEPYPMDLLLLADPSEELVREYIERGRCFTGAAADGRVIGVYVLLPTRPGTVELVNVAVSEDCHGRGYGKEL
VLHAVGQAREFGFKTIEIGTGSTGVAQLALYQKCGFRITGIDKDFFVRHYAEEIYENGMQVIDMVRMSQDL
>lcl|NZ_CP026520.1_prot_WP_042227129.1_3918 [locus_tag=PC41400_RS19800] [protein=hypothetical protein] [protein_id=WP_042227129.1] [location=4418421..4418660] [gbkey=CDS]
MAESENEIQLSEEEVAAGEVEAVEEVQADAEAETAEDADSDADTLEDSTEADDEEVTETVESDESDEAVEDTASEDVTD
>lcl|NZ_CP026520.1_prot_WP_042227131.1_3919 [locus_tag=PC41400_RS19805] [protein=DUF6509 family protein] [protein_id=WP_042227131.1] [location=complement(4418907..4419200)] [gbkey=CDS]
MLTITAYSAELVKDPFGIIPGKRYEFNLEVDVPEDDELYTANGIYLRVIYGVDEGQSRIIKYDIRENSTDTYLEFDLEDE
ELAVVDTFCKEHLPQDE
>lcl|NZ_CP026520.1_prot_WP_164977144.1_3920 [locus_tag=PC41400_RS29905] [protein=hypothetical protein] [protein_id=WP_164977144.1] [location=complement(4419288..4419443)] [gbkey=CDS]
MKIMITQGEALEKGIWPEVMKMFGLNQEDDVWANEEFVLSEEQAKTLGLIS
>lcl|NZ_CP026520.1_prot_WP_042227133.1_3921 [gene=odhB] [locus_tag=PC41400_RS19810] [protein=2-oxoglutarate dehydrogenase complex dihydrolipoyllysine-residue succinyltransferase] [protein_id=WP_042227133.1] [location=complement(4419824..4421098)] [gbkey=CDS]
MADIKVPELGESITEATISKWLVTEGSTVSQGDLLLELETDKVNLEISAEQDGVLSEIRKQEGDNVSVGEIVGVIGASSG
APASAAPAAAPEAPKAAEAPAATPAAEAPAKAPAASAAASDSASLAASPAARKLARERGIDLSQVKSQDPIGRVYQDDVK
SHGTQAAQPASAPAAAKSGGAGQAAQQAVPGKPVERKRMSRRRQTIANRLVEAQHTAAMLTTFNEVDMTAILDVRKRRKQ
AFLDKHDVGLGFMSFFTKAVVGALKKFPHLNAEIDGDDIVVKQFYDIGIAVSAKEGLVVPVVREADRLSFADIEKQIGDL
AGKARANTLALSDLQGGTFTITNGGVFGSLLSTPILNAPQVGILGMHKIQIRPVAIDAERMENRPMMYIALSYDHRIVDG
SEAVRFLVTVKELLEDPESLLLEG
>lcl|NZ_CP026520.1_prot_WP_042227468.1_3922 [locus_tag=PC41400_RS19815] [protein=2-oxoglutarate dehydrogenase E1 component] [protein_id=WP_042227468.1] [location=complement(4421124..4423973)] [gbkey=CDS]
MRNEVEGSRWPWESYYGPNLGYIQEKYEQYLNDPDSVEAAYRDLFARFGAPPASVQTVQSEGQANAGTTTTSDPNVLKKV
VAVGKLVWNIRTYGHLAANLDPLGLSTNPDTRLLEPGTYGLTKEDLVAMPASIVWEEAPSDVANGWEAIQRLRQRYSQSI
GFEFSHVHEEDERNWLNRQAESGASIEPLTKEERTKLLDRLVQTEQFENYLHRTFVGQKRFSLQGTDVLIPLLDEIVHEV
VHDGAEHILMGMAHRGRLNVLAHILGKPYKKIFAEFHHAPNKDIIPSEGSTGINYGWTGDVKYHLGARRSIKEGETREAK
LTLANNPSHLEFVGAVVEGFARAAQDNRENPGYPEQDLNKAACIVMHGDAAFPGEGIVAETLNFTKLTGYENGGTIHIIV
NNRVGFTTDSTDSRSTHYASDLAKGYEIPIVHVNADDPEACLAAARLACEYRNRFKKDFLIDLIGYRRYGHNENDDPETT
QPLMYGKVKNHSTISVLYAEKLKQEGLVDDNGYQQMIQNIQNTLQEAYELVKNHQGKGKRVAEPAIFDDLKKIKTAVPLQ
ELQEINRELITWPEDFNVYPKLQRILERRGDALNEGGKVDWALAETLAFASILADGKPIRISGQDVGRGTFAHRHLVLHD
VKNGSTYSPLHKLPQARASFAIYNSPLSEGSVLAFEYGYNVYSPETLVIWEAQYGDFANAAQVIIDQFISAGRSKWEQKS
SLVMLLPHGYEGQGPEHSSARLERFLQLSAENNWTVANLSSAAQYFHILRRQAAITEREEARPLVIMAPKSLIRNARVSS
DGKEFSESSFLPVMEQKGLGNQPKKVKRLVLTTGKMAIDLEEAFETSEDRDFSWLHIARVEQLYPLPEQDIAEIIAKFPG
LKEIVWVQEEPRNMGSWNFMESRIRSLAPDKVNVRYVGLPDRSSPASGFQEIHTFEHNQIISLALKQPE
>lcl|NZ_CP026520.1_prot_PC41400_RS19820_3923 [locus_tag=PC41400_RS19820] [protein=MBL fold metallo-hydrolase] [pseudo=true] [location=complement(4424281..4425102)] [gbkey=CDS]
MENTFEDKRLPVTSVKSGEGLEAAPDIYCLTLQIVNVVFVGEVDNPQGWVLVDAGMPRSADEIIQAAEKRFGTVPPRAIV
LTHGHFDHVGAIIPLLKR*DVPVYAHELEIPYLNGMQDYPPADPDADSGLVARMSPVFPNEGIDLGSYLRKLPSDGSIPE
MQDWRWIHTPGHTHGHISLFRDEDKALIAGDAFVNVKQESLYKVIMQTPEVNGPPKYFTTDWEAAKISVRFLEGLQPSLA
VTGHGRPVSGQELAEGLRTLSEHFEERAMPEKK
>lcl|NZ_CP026520.1_prot_WP_042227134.1_3924 [locus_tag=PC41400_RS19825] [protein=spore germination protein] [protein_id=WP_042227134.1] [location=4425313..4426791] [gbkey=CDS]
MRFKKWMRLLSSTNPPSPVIRQIHAEPPVLSRIKEQFKDCSDLTQREFPELEITLFYFKHMVDPDLFEREILAPLENIQA
DETDELLQNSIFTRVTEERVLVKKILSGWAALFVQGTAYAVDIYGPEGRGIEQSETEGVIVGPHDAFTESPEANLSLIRR
RVKSNHLKVQTFTVGEITSSKVFVLYIDGIANMDYVNTMTQRIEAIETDAIFDATMLVQLIDDFPNSVFPVMLNTERPDV
AASKLVEGKVVAIVEQSPSVICGPTSFFEFFISPDDYYSRWAVGTATRLLRYAAFVITISFTALYVSVTTYHYEMIPDNL
ITTLTESRSRVPFPPVFEALLMETTIELLREAGARLPTKIGQTIGIVGGIVIGQAAVQAGFTSNILIIAVASSAIASYVI
PSYTMSASIRLIRFGLVIAAGVLGNFGLMAGIALVIIHLAKLTSLGAPYTMPVAPLKLKDWKDIFIRAPFWSLKSRPSQS
LSMNKKYNKFKK
>lcl|NZ_CP026520.1_prot_WP_241688704.1_3925 [locus_tag=PC41400_RS19830] [protein=GerAB/ArcD/ProY family transporter] [protein_id=WP_241688704.1] [location=4426792..4427883] [gbkey=CDS]
MVALSRERLYTHHVGFLVFNTQMGVVLFSLPRIAATYIGTNGWASFLIFSVISTFNIYLISLVNKLGKGESFFDIVEKTI
PKWVLAPLYLFMTSVFGMIGCLVIKQYVLIYQLLIFPSTPDIVLKLVVDLLVLLLLLKGIYTISKANIIFSLLLLLLVPI
FFAFTTEFDWVRLTTSLFKDGEHSVKGFIKMYSAFMGYEISLFFMPYASRSVKWMRGVYIGNFVVTFVYLMVTFICYGVF
SYVQLKLQTFPLLDLLAYLKFPFMERVQNFLYSVFLFSVLVTAAMYFWSSKETLSRLVPKIKGTYLSPAIMVATLFISYM
AQTLTVVEKWFGYLSVMLIGLAFFLPVFALVVLLFRKRSLPNV
>lcl|NZ_CP026520.1_prot_WP_042227137.1_3926 [locus_tag=PC41400_RS19835] [protein=Ger(x)C family spore germination protein] [protein_id=WP_042227137.1] [location=4427876..4428973] [gbkey=CDS]
MFKMRRLFLPSLLFLSGCSSDQNILEKIGYIQSAAYDWAPDNKMKVTLSIPVIERESRDGYDEILTITASSSKAAKNLLS
RKTSRALVSGQLRNLLYGIELAKTGLRRNMDSILRDPSISKRSNIILTEGEAHDIINRDYTGHPKTEQYVDKLLQKEFQL
QNTPRILLHQFVRDYFDEGIDPIATVIRIAGNDLEVSGIALFKEDVYVTKLPTKDMIFFNLLYQNQKQGEFSFTLNDEET
KSIMFNAISSKRKIKIRRDEEGVLHADIRMVVHGSVLEYIGKLNLSTDDSAKVEKKMSAYISEHSERVLSFMQKHSVDSI
GLGRYVRNELGYAEWEKLDWAEAYPSVQVHVNAEFHIKNFGNFFD
>lcl|NZ_CP026520.1_prot_WP_042227139.1_3927 [locus_tag=PC41400_RS19840] [protein=metal-dependent hydrolase] [protein_id=WP_042227139.1] [location=4429175..4429822] [gbkey=CDS]
MKGSTHLAIGTAIGLAASAFHPFTPTNFAVYISVAAFSALSADLDGPSMLSSKLGKISKWLREVFLWGGALLVAAVAYLF
VTKHQFYPERMTIAVMIFLIGFVTKEGIIRNALVSLVGAGLIYWGWNTQMRWLIGLGAFVVWAPWLKHRGMTHTVWALAL
WGAIGMELEKQLQLEGIMMVAVAGYFSHLFADTLTPSGVKWLYPLYKKSFKIPFL
>lcl|NZ_CP026520.1_prot_WP_042227140.1_3928 [locus_tag=PC41400_RS19845] [protein=sensor domain-containing protein] [protein_id=WP_042227140.1] [location=4429897..4430562] [gbkey=CDS]
MNKSPDLTASFFYLLALLPMGLFGFVYAVVGIALSIGLMPIFIGFPLLLFVLRTNRNLTELDARLASRLLGLEPQTEAAA
SGGTLRGERKSFWAMLGAELVDARSFSALLWQFIRLPLGILAFAFTVSIASVILALLVSPVVYWILLQTLHIDIFHDQLN
FFLMNERIDSMQLSLIYTGLGILAAPVGMIAIRKLAQFMGSIAVEMTRIGAPAYQPAPQYR
>lcl|NZ_CP026520.1_prot_WP_042227142.1_3929 [locus_tag=PC41400_RS19850] [protein=hypothetical protein] [protein_id=WP_042227142.1] [location=complement(4430603..4430794)] [gbkey=CDS]
MEVNGENGIGWLTLALINAGLAQGKNRSGLNWFFISLLIGPLATLLIVVWDRIPKEPERKKMY
>lcl|NZ_CP026520.1_prot_WP_174446786.1_3930 [locus_tag=PC41400_RS19855] [protein=protein tyrosine phosphatase] [protein_id=WP_174446786.1] [location=complement(4430962..4431297)] [gbkey=CDS]
MKLLFLCSRNRWRSLTAEKVFNEVKSWQVRSAGTEQNARIKVTEGHIGWADVIFVMEKKHKRRIEDKFGSVVRQKRVICL
EIPDDYEFMDEELIELLESRVSDYFPLTEDY
>lcl|NZ_CP026520.1_prot_WP_042227147.1_3931 [locus_tag=PC41400_RS19860] [protein=hypothetical protein] [protein_id=WP_042227147.1] [location=complement(4431293..4431841)] [gbkey=CDS]
MTTIQSIVGLFLIVFGTFISFSTGNLPAAFISLVGGFILMSLSKLVDLQQGAYLKALGVPITGDQFQLIMKYSPEYAVES
ADFNVYPESHKEYALIHLEGELYLSVQVFKNVMTRVENEYTFRFPGREPVSFHRTLSLYKGAELFEYGGLAFVRISALNL
VGSICEDRLLLNFAVQGRETDN
>lcl|NZ_CP026520.1_prot_WP_174446779.1_3932 [locus_tag=PC41400_RS19865] [protein=AAA family ATPase] [protein_id=WP_174446779.1] [location=complement(4431855..4432445)] [gbkey=CDS]
MIIWVNGAFGSGKSQTTHELHRRIPGSFLYDPENAGYFIRKNVPPSTALDDFQDYPMWRESNYGMLNYISGKFDGVILVP
MTIVNPVYFDEIVGRLRRDNRTVHHFALCASKETLLKRLRSRGEGPNSWAARQIDRCRGGLSQEVFQHHVDTDFLTIGEV
SEGIAAIAGIQLLPDNRSAIRKKADRLLTQLKQIRF
>lcl|NZ_CP026520.1_prot_WP_042227150.1_3933 [locus_tag=PC41400_RS19875] [protein=pyridoxamine 5'-phosphate oxidase family protein] [protein_id=WP_042227150.1] [location=complement(4432757..4433380)] [gbkey=CDS]
MNYFEDRIEAEAELREMLGYPNKIVEAMKITYMERHCREFIGLSPILFASTADLLGNCDVSPRGDAPGFVKVLDDHHLII
PERPGNRRMNTYRNILENPHIGLIFIIPGLEETLRINGKACLTKNPGLLAPLEAHGKIPKLAIGVEVEEVFLHCAKAFKR
SELWDTKTWPGALPSVPEMIAAHAGRPDITADVVSEGLKESYEKRMY
>lcl|NZ_CP026520.1_prot_WP_042227152.1_3934 [locus_tag=PC41400_RS19880] [protein=erythromycin esterase family protein] [protein_id=WP_042227152.1] [location=complement(4433523..4434959)] [gbkey=CDS]
MYSHHLKLKTVKTTIALTIAAVCAFPAAGMAAADTAGDSKPPAVSTTEQYGRNWANWIQSHAYALDRIQPETSVKGVIEK
DRFKDLEFLKPLLIDKKLVYLGENTHGAAEYSSSKVRLIQYLHQELGYDVIAFESGLGNASAALAKSADSTPEQMMKEAI
FGVWWSKETLPLFDYIKQTLATDKPLILSGFDMQIQSPYSEFVRDWIGSRDAKLAGTFADAEQELGDWSFSEDEAGYAKV
KPRLLETYESMKTFVKENADKLKADYPRNPHLIEMTQRVLDDRIEVIRTYTEANIRSNIALKKNDISPFRETVRMRDEMM
ANNLTWLAEQIYPDKKIIVWGHNVHIRKKNSAVLNSGYTGLSLMGESMPARLKRQSYVIGLYTYQGEAANNMGQSYPIVK
PERGSLEDILKQHGHPYTFVDIKYRKDKPGTSWMFEPRLSLDWGLMQESFIPRDQFDGLLLIDTVHAPSYMRGKPGSQ
>lcl|NZ_CP026520.1_prot_WP_042227154.1_3935 [locus_tag=PC41400_RS19885] [protein=hypothetical protein] [protein_id=WP_042227154.1] [location=complement(4435085..4436521)] [gbkey=CDS]
MSDDEPDDLQETGSLKKAEPDGISAGSDPGEEQAPPAHEAVTFWKLLVFFVPLGIAASLVTMSHSIINSTLARAPDPEHV
IAGYAIAMSLLGITEKPAVLLRQTCSALVRDRVSFRAMRRVAFYVFASIMAAGLILSYLPPGKWLMGAVFGIGGAQLENV
LNVYRILMFVSLFSGLRCLYHGLIIYNKRTFWLTIGMVVRLVGMYLVAQYYIRVHGVTGSSVGAIIFLVGMVIESAVAVW
EGSALLKHKIPEKNPEHTVEKTGDVFRFYKPLLFSSFIAVVIGPSLNALLGKAGDAVIAIAAFAIASNLTQLVQSFFSYI
HQIVLNFYRIHPGRVKRFVLVISFIPSVLIGLLAYTDAGPWFLNEVIGVGDRLMEEVIRTLRVFMILTLVFPWLDYGNGI
LMLRRETRTMVYSQSANVTVMLVSLVLLIVITPGWNGAIGAFAQSVGTLAECIVVFAAVKRGGGEALVSGSAPREKSV
>lcl|NZ_CP026520.1_prot_WP_241688705.1_3936 [locus_tag=PC41400_RS19890] [protein=MMPL family transporter] [protein_id=WP_241688705.1] [location=complement(4436723..4438858)] [gbkey=CDS]
MGLQRLAVLSFRYPKIILVCWAACLVFFGGYAHSLSSVLKGHGLDPEGSFGAVRQRLDAGFNRPAEPVVLWFEQNAPGPK
EEFKAYIRSVLRSAEGLPGLERIVSPLDQGEMLRSGMAYALLDIRFTGRDEQAVLSELRGRLPVHPGFAWKLTGKPVVQA
DVNTASEKDMRKAELIGLPAAFLILWIAFGGAATALIPVVLGLVSAVVSMGIVYWIGMVTDLSNFVLNVIPMVGLALSID
FSLMLVSRFREELICGRAEPALRLTMRTSGRAVLYSAGCVALGLLGILFIPLPMFASVAWAAFTVLIVSVFLSFTLLPAV
LSLLRTRIEAENAARFHGNGGRFWHRLSVGVMKRPLRMALLAILIVFICVLPVGRMKIAVPDAASLPAGTESREAAERLA
ARFSSPEVTLLLGAKDAGPHSGQPFDWQHAYRLFQQLQRDPQVVRVDSIFSRLGMNGEQLRILYGEKPGAGNNAGAASFY
FHQNQLVMDVYLRTDPDSEETRQWLRKWEAAGNQERIPFRLGGESKYRQEVYDAIFTNLYKVLLFVLVTNFFLLYKAFGS
VFIPLKTLVMNVLSITASFGLLTWIFQEGAVDGGGSIAIMIPVFIFGLVFGISMDYGVFLVARIAEVYRQTQDNDYAVAA
GLASTGRIITSAAAIMIAVTLPFAGADVAGVKQLGTGIAGAIFLDATLIRMILVPALMKLLGRWNWWSPYN
>lcl|NZ_CP026520.1_prot_WP_042227158.1_3937 [locus_tag=PC41400_RS19895] [protein=NAD(P)/FAD-dependent oxidoreductase] [protein_id=WP_042227158.1] [location=complement(4438891..4440066)] [gbkey=CDS]
MNVTETDVAVLGAGLAGALMAKSLAEKGWRTLLVDRHRFPRHKVCGEFLSPEVRQTLDAAGLLDRVTALRPARVERARLI
SADGGYVDIPLPGTAMGLSRFRLDEALHEAAAAAGARVLTSAAVLSVTRLADESCEVRVRREAGMEVIRARTVIAAWGAN
GQARIAKAGRTRRTLNPWKRSAPDYMGVKSHYTGIEMEPIVELYLFPGGYLGLCPIEDGLVNVSALLSRESFRGAGKSVL
ALLEAAARRCPPLHSKLKRGLPVPDAQAAVAPVKLDRRLNPWDMMPRLGDSSAMIPPLCGDGMSMALRSARLCAPLANDY
LRGKLSIAEWEEQYVRAVKRTFGRPLRWGRLLQSSLNSPAAVRLLLRLAQLSPGMASGLLEATRLKPERES
>lcl|NZ_CP026520.1_prot_WP_042227159.1_3938 [locus_tag=PC41400_RS19900] [protein=methyltransferase domain-containing protein] [protein_id=WP_042227159.1] [location=complement(4440063..4440809)] [gbkey=CDS]
MPPFDRLRSRSPEAEYMDDLTCGGAELTEALRHLRRLNRLFAASGPTLYGIEKLWTAAGKPAELTVLDVGSGSGDVNRRV
LRWADDRQIRMTVTLCDITEEACEEARRLFHDEPRVKVRRTGLFELPEGSADIVTGTQFLHHFSSDELPGAVRAMLRASR
LGIAVNDIHRHWLAWSAVWLASRVLSGNRYIRHDGPLSVAKGFSPEDWSILGKALETPGYRVSWRPLFRYAVVIPRQPQP
DAGSERKP
>lcl|NZ_CP026520.1_prot_WP_042227161.1_3939 [locus_tag=PC41400_RS19905] [protein=type III polyketide synthase] [protein_id=WP_042227161.1] [location=complement(4440796..4441944)] [gbkey=CDS]
MNQTTPATAILGIGTAVPAYPLKQSEVVLRLTQALERYPDSARWAKRLFKQCGVETRYTCEPGLLGPVSSCRYLPDTEPG
LIPTTAERMALYKRESVPLARQAAGQALADSGLSVSDITHLITVSCTGQFLPGLDASLVKELGLSREISRISLQFIGCAA
GLRAVCLAREIADSAPQANVLIVSVELCTLHIQPSGDKEALFGASFFGDGASACVIGRTGKSRQGIFGLGNHRSVLIPEA
AEEMTWEVGNFGFDLYLSPAIPKLITNRIPQEIGKLLDGERLPGLWAIHPGGKGIVDAIEQVYGLTEEQTAPSRQVLRDF
GNLSSATILFVLQQMRMQLRTAGADAADGIALAFGPGLTAELIRFTYIPVGAAQEKTAHAAV
>lcl|NZ_CP026520.1_prot_WP_042227162.1_3940 [locus_tag=PC41400_RS19910] [protein=YdcF family protein] [protein_id=WP_042227162.1] [location=complement(4442299..4442883)] [gbkey=CDS]
MTKKQIKLILAFGLFIVLMIAVKNAGSFLVYTEEPVKSDVILVLSGDRGERTEKGAELYKNGAAPYLMFSGGEVYRNNRM
ADLMKEHAMELGVPEQAIVLEDRADSTYENAVFSKTIIEKHAFKSVTVVSSDYHMRRVKLLFDRAFKDSGIQLHYVAAPD
KNFTPRKWWGNNKSLMVTFTEYLKLTGYFFGKNS
>lcl|NZ_CP026520.1_prot_WP_042227163.1_3941 [locus_tag=PC41400_RS19915] [protein=MFS transporter] [protein_id=WP_042227163.1] [location=complement(4442984..4444255)] [gbkey=CDS]
MNTREMTPPEELPLVRQDQDTARTAGWLLIVGILFIGAVLRAPLTSVGPLIGTIKGELGISNTLAGTLTTLPLLMFAVIS
PFAPKWARKYGIETVLLAAAAILTVGILLRSASGAGTLFLGTALLGVAIALCNVLLPSLIKRDFPARIGLMTGVYSVSMN
LSGAAASGVSVPAASSLGLGWKGALGAWAALSLLALFFWLPLVTSSRKARQVARVSTVSANSSGEKSVSVWKSPLAWKVT
LYMGLQSTFFYAFVAWLPAVLNTRGLSAEYAGWMLSLMQLALLPITFVVPILAARMANQRFLVTLTCVLFLAGILGLLLG
STALVPLWIILAGIGGGCAFSLAMMFFSLRTRSAQQSAELSGMAQSFGYLLAAAGPILFGFLHDATGGWTTPLLLLAVIA
VLLFIVGQGAAANRFVGEERGAA
>lcl|NZ_CP026520.1_prot_WP_042227165.1_3942 [locus_tag=PC41400_RS19920] [protein=FadR/GntR family transcriptional regulator] [protein_id=WP_042227165.1] [location=complement(4444252..4444944)] [gbkey=CDS]
MLEKPKKLTLVEQVADQLQTLIESGQWPIGTRIPPEPELSETLGVSRNTIREAVRAMVHAGLLQTKQGDGTYVCSSSVLG
VVLQKRVRQSTLAETLEVRQALEREAARLAAKRRTDEDIDSLKLHLDACAAAEESGDMEAYSGADIRLHQAVVTAAHNEL
FSELYAHMTEALDHSVRSLSGLIPEIGLYRRSHAKMIDAIIRQDVEEAVEAVHEYISHSRNDLKRLEEQS
>lcl|NZ_CP026520.1_prot_WP_042227166.1_3943 [locus_tag=PC41400_RS19925] [protein=MFS transporter] [protein_id=WP_042227166.1] [location=complement(4445303..4446517)] [gbkey=CDS]
MRAIAMLQNPKQRKLLFSAGLSWLFDAMDVGIISFIVAALTVHWKLSAGQVGVLTSLNSVGMAFGAAIAGYLADRYGRKS
VLVWTLLIFSAASGLSALAASFTVLCILRFIAGVGLGGELPVASTLVSESMPVQDRGRAVVLLESFWAGGWILAALIAYF
VIPKYGWQSAFVIGAVPALYALYLRRAIEDPPRFRKERSGESSFGAKVKSVWSRDYRRSTVMLWVLWFTVVFSYYGMFLW
LPTVMVLKGFTLVKSFQYVLIMTLAQLPGYFTAAYFIEKFGRKFVLVTYLVLTAFSAIWFGGASTEGMLIAAGMCLSFFN
LGAWGGLYAYTPELYPTAIRSTGAGLAASFGRIGGIVAPLLVGVLVGWGIGIQAIFILFFVMILIGALAVLFWGDETKGK
ELSS
>lcl|NZ_CP026520.1_prot_WP_042227168.1_3944 [locus_tag=PC41400_RS19930] [protein=DUF1003 domain-containing protein] [protein_id=WP_042227168.1] [location=4446722..4447285] [gbkey=CDS]
MSSKSETERTSEERLLKELEKYPRQKLHKEDVERVTEMVRDYQGKIKAHLHEQQEKKTNHWDRMADHVAVLAGSWWFVTS
SAVIITVWMLVNSERNSLFILSFSLSILSLFQSTLIQKSQNRQAAKDKQEQMLDIAINYKAEQENLEIQQNLKELEVKLQ
WIELRLSARSRAKSRPGRTKTPRTRRK
>lcl|NZ_CP026520.1_prot_WP_042227170.1_3945 [gene=glsA] [locus_tag=PC41400_RS19935] [protein=glutaminase A] [protein_id=WP_042227170.1] [location=complement(4447292..4448236)] [gbkey=CDS]
MNAGHEFSAVRAALPQWVEESRKLTSRGNVASYIPELAKSPASALGIHLLNGQGEHLTAGECGLPFTMQSISKVFTLILA
LMDHGEATVFSKVGKEPTGDDFNSMLKLELVEPGIPFNPLINAGAIAVSSLIKGDGPEEKSARVLSFFRELAANDKLDYD
EAVYRSENETGHMNRSLAYFLKENEVLDGCVEEVLQVYFRHCSIRVTCGDLSRMALVLAFDGQDTLTGKTLIPRRFVQIA
KTFMTTCGMYNASGEFAISVGLPAKSGVSGGILTLVPGRYGIGVIGPALNRKGNSVAGVDLLEKLSGRFDWSLF
>lcl|NZ_CP026520.1_prot_WP_042227172.1_3946 [locus_tag=PC41400_RS19940] [protein=GAF domain-containing sensor histidine kinase] [protein_id=WP_042227172.1] [location=complement(4448367..4449635)] [gbkey=CDS]
MTNRHQDPSGFYNNIEEAAAHIIEVVSGLLQVNTIFVATNDGRTNTILKAFNRREQLVEEGDELPFDISYCSLVGRRAGE
PLVISDTASSPLTADMGVTRALGSRSFIGLPIILKNGHFYGTICILDHPSYPVSETDLKTLSAMAVFLAHVVELEHAVAE
LRDAQQILRESHAASEQSSKIKSRLLAMIGQEIKTPMTGIMGMTDLMQDTELSEEQKSYMNIIETSNASLLNLVTDILDY
SKMESGGVTVEAEPFELLPVLEEVVNLYENRNTGARKVRLDAEPGLPSVVVGDPVKIRDTLVKLLDHASKNARSGELTLA
AVNLPDTDHPDAHPIRFAVKAPLITPDKLEFLLETVDGKGGESSVPVYETSLDLAISKKLVELMEGRIWTESDRQKETDL
IFTLMLNKYAETAEAAELNDFK
>lcl|NZ_CP026520.1_prot_WP_042227173.1_3947 [locus_tag=PC41400_RS19945] [protein=aromatic ring-hydroxylating dioxygenase subunit alpha] [protein_id=WP_042227173.1] [location=complement(4449823..4450899)] [gbkey=CDS]
MNLTKNDVCQSGDGKEALYAELEKHWLPLCPSRELKRKPLGLTVLDVPLVLFRTEEGAHALLDRCPHRNVPLSGGRLTAG
KLVCPYHGWEFDGTGACTRVPGLCTFKPGPHHDASPAALRERDGFLWVRLGGEARSGEPQEASVPEPLALPFMNNPEYYS
FVWKTAVRGSLLNAAENLLDATHTHYVHAGLIRTDGGRQTVNARIRAEGASVEIEYTGESGQAGLISRLFERDRQRSFGR
FYMPGVAQIEYSSAKGLTLLITAVMRPTSFAYQDIYAVITMKRGRFTNGLKKRVLTPFLKKALNQDITIVETQQRTIERW
GGESFHSTQADLIRPYLEQLMKGKVYDTPHEKKVQLFV
>lcl|NZ_CP026520.1_prot_WP_042227174.1_3948 [locus_tag=PC41400_RS19950] [protein=GNAT family N-acetyltransferase] [protein_id=WP_042227174.1] [location=complement(4450896..4452089)] [gbkey=CDS]
MTQPAESIERQAGQTPRSGAGAEGSPSNETGQCKNHPANKMTVAYTKPLIGKGADAYIRNVRTKVELVRAGEQLLPVTVN
ETEYDNSYVCSPYSTYVSYALEELVLFKSELLKGVLRGVIGILSPLLKRARINRNVHVNNWLLSTNLYEELPKEDVRVLT
ADLQERFPSHAVIFRSLNEVTNPQLMKDLTGLGWILVPSRQVYFFEGREPDYLHKPNNVWDQKLLRRSPLRIVEHEEIEP
SDYPRIVELYEDLYIRKYSRLNPRFTEAYIANCHLHGLLRIRGLRDSSGVLQGVIGCFDKAGVITAPLVGYNTALPQKEG
LYRMLMSLVLTRAAEEGKLLHLSSGAAHFKRIRGGRPAIEYSAVYIRHLPLRRRAVWLGLSHLLNGVGVPVMRKYRL
>lcl|NZ_CP026520.1_prot_WP_042227176.1_3949 [locus_tag=PC41400_RS19955] [protein=SMI1/KNR4 family protein] [protein_id=WP_042227176.1] [location=complement(4452109..4452669)] [gbkey=CDS]
MQDYFFDEVFVKDAGAAEGILESFHQTWKQPLTEEEIQEIIARQKNPFPTNSPYYNQYIPIDPTKWRMPEGELPGAYAEF
LRYSDGGEFRTGERYFQFFSTSDFREMNLAYELPEYMPLSVSFAMDGCGNHYLFDMREAPVEGEYPILAAHSGNLGYDEC
VRVADSFEALCRGKRSLDEELDSRYT
>lcl|NZ_CP026520.1_prot_WP_042227177.1_3950 [locus_tag=PC41400_RS29910] [protein=ATP-dependent carboxylate-amine ligase] [protein_id=WP_042227177.1] [location=complement(4452738..4454033)] [gbkey=CDS]
MNVLLTGCRSYTALDLARRFRDSGWNVFAADSVKFPVCRGSRAVKEFFHVPSPREGCREFIAALAAIIRSAGIDLLLPTS
EEIFYIAHGREELSRCCRVFCDDFEVLNSLHNKWAFSGLTEGMSARTPETKLLTSAAQLKELGDPSAFAFKPVYSRFASE
ALLAPSRLPEGFAPSPDRPWIAQRYIRGLELCSYHLAVDGEIAAYGSYEPSYRLGQASGYYFKPVHREKIDRFAAELVRK
LDFTGHVAFDYIEEKDGTLHVLECNPRATSGLHLAGTAGLMKPFGAGGERLPEVTGGKLSAGGRTPLENPAAIRISRGSS
RSSGDPGNPGSSGPQGNARNPGNPVPAEAPPMIGYAMLANPFYHGFSLSGMGRWFNDFRHGRDVLFSRQDPWVPLLHTRS
LAETALDSLCRRIPFKDAATADIEWDGEALN
>lcl|NZ_CP026520.1_prot_WP_042227179.1_3951 [locus_tag=PC41400_RS19975] [protein=adenylate cyclase] [protein_id=WP_042227179.1] [location=complement(4454030..4455385)] [gbkey=CDS]
MKLILLLKQYALTKWNGQRWKSREDLEAWQNARVRAFLEETRRRSPFYARLFEGLPLDNWREWPTIDKTAMMDAFDELNT
AGITKDEALQAALQAERERDFSPELNGVTVGLSSGTSGNRGLFIVSAEERAAWAGAALAKALPGSLLRKHRIAFFLRADS
NLYQTVRSRRIRFSFFDMIRPVPEHIEQLGWLQPTLLIAPPSMLRLLAEAKQRGELAIAPDKIISVAEVLEPVDRQVIEL
AFRTRVHQIYQCTEGFLGATCKHGTLHLNEDIAVIQREYVDAERRTFMPVITDFSRASQPIIRYRLNDLLTLRDEPCPCG
SVFTAVESVSGRTDDLFFGRSMDGGRDVPVFPDFIRRAVMMASDGIAEYKVIQRDRERLEVSVRTKEGVPDVIGPLRREL
AGIFAQHGCRAPEIMFTPYSFRPGLTKLRRIERAYAVEPDLTARDGWIGRT
>lcl|NZ_CP026520.1_prot_WP_042227180.1_3952 [locus_tag=PC41400_RS19980] [protein=MBL fold metallo-hydrolase] [protein_id=WP_042227180.1] [location=complement(4455382..4456227)] [gbkey=CDS]
MKLTMHCTGACKQLEILSLRGGKWRTVAFPGLFAMLEHPSKGLILFDTGYSPRFAEATRRLPYALYRWTTPVETGVHLTA
AEHVRAAGYAPEDVTAVVISHFHADHICGLADFPNAKFICSQAGYDSVASLKGLAAVKRAVLPDLIPPDFKARAFWVEQC
PEAALPAACRPFERGWDLFGDGSVTAVDLSGHAAHQIGVFLRGADGNPVLLGADASWSAQAVRENRLPSPPAFILFDGRQ
DYKRTFGNLVELHRRQPEMRLLFTHCAEAWELCGKEGIELL
>lcl|NZ_CP026520.1_prot_WP_164977145.1_3953 [locus_tag=PC41400_RS19985] [protein=NAD(P)-dependent oxidoreductase] [protein_id=WP_164977145.1] [location=complement(4456224..4457228)] [gbkey=CDS]
MVTMKALVTGATGFLGGNLIRRLLEEGYEVTAVGRNPEAGKRLTQEGAGFVRMDLASDDMPEVVRGHEFVFHCGAMSSPW
GAYRDFYTSNVVGTRQIIRGCMKHGVRRLVNVSTPSIYFDYRDRLGISEEEPLPERGVNAYAQTKRQAESELDAARKAGL
PVITIRPRGLFGPGDTAIFPRLLRANEEKFVPVFGGGRAFMDVTYVDNVTEALLYCAHADKACDGQKYNITNGEPVYVYE
LLAELFRLLEMPFRARKVPYPAAYGAAAAMEAFAALPFARKEPMLTRYTVALLAFSQTLDISKAERELGYKPRVSVADGI
RRFSLWWREQEGRA
>lcl|NZ_CP026520.1_prot_WP_042227184.1_3954 [locus_tag=PC41400_RS19990] [protein=ATP-grasp domain-containing protein] [protein_id=WP_042227184.1] [location=complement(4457143..4458393)] [gbkey=CDS]
MSTRKKLGSRRILITGGRAPAALDLARQLSALGHTIYAADSLASPLLRFSRCIEKFIRIPGPRDNASAFAEALQAIVRER
SIDLLIPTCEEIFYVAQIKSKLQGCTVLTDDFGKLIALHDKASFIRLVEEMELPAPHTVLLETERDMHAFLEREGTRKWV
LKPVFSRFANHTRLYGPGEALPRINVSPDNPWVAQHFVEGREYCVYAVAHKGRLTAYADYAADFTAGRGATIYFKSCGRR
DLFGWVEYFVREIRFTGQIAFDFIVHESGVIYPLECNPRATSGLHLFAPEEGIDRALLDCTDTVIIPSPGGRPAMITAAM
LMYGAAACGTPAGLRQWIAGLARGRDVIFSRKDWGPALAQGKILLDLWREARAKRVPLKEITTYDIEWNGYYESVSDGGD
RFSRRKSDPPPAGGRL
>lcl|NZ_CP026520.1_prot_WP_042227185.1_3955 [locus_tag=PC41400_RS19995] [protein=beta-ketoacyl-ACP synthase III] [protein_id=WP_042227185.1] [location=complement(4458383..4459381)] [gbkey=CDS]
MKTRQVKVLGTGAYLPEKRVTAEDMDRRLGKPEGWVYKKSGVKVRHYAENESVSQMGAKAAFAALADAGLEAERLDCIIC
TNAVPEQPIPCTAALIQSEMKLGDSGIPCLDVNSTCLSFLAGLDLISYLIDAGRYDRVLLVAPEIASVGLNYSQSESAIL
FGDGAAAVVVGRTPAGESSGIVASRIETYGSGAALSEVRGGGSKLPSIAYEPERIADFLFDMQGEALFKKTSKLLPDFLK
KLFGEGGVTLEDVALVIPHQGSAMAMRLLQRKLRIPDDKLMNIIEQHGNMIAASLPMGLHEAIRLGRLKRGDLTAVIGIS
AGLSLGGLLLEY
>lcl|NZ_CP026520.1_prot_WP_042227186.1_3956 [locus_tag=PC41400_RS20000] [protein=DUF6492 family protein] [protein_id=WP_042227186.1] [location=4459693..4460505] [gbkey=CDS]
MSEVSGTGRTPIDVLIPAIDKDLETLPFVIDGLRRHVQHPIGSIYIVSPDSPRIKKLCKDKKCKFVHENSVLPLTKKDIR
YRSNRWDRSGWLYQQLLKLNGDKICGNRYFLVTDADTVLIRPHRFLENGKQVFFTRNWSHQEYFRIHAKLLGQKAVSRPS
YVNHYMLYDKVKLSRLKKAITARHKMPWYRAILKNLNTTKQFGFSEFETYGNFVRAQYPGTSLLKRSLNKEFKKSIRFVP
HSEREKWAGKFRSLSFHKRGGYYFKAKKAK
>lcl|NZ_CP026520.1_prot_WP_042227188.1_3957 [locus_tag=PC41400_RS20005] [protein=sugar phosphate nucleotidyltransferase] [protein_id=WP_042227188.1] [location=4460564..4461979] [gbkey=CDS]
MKIILLSGGSGKRLWPLSNEIRSKTFLKLLPAPDGGRESMLQRVCRQLDAAGLLSSTSIVTHFSQFEIVRSQVGEKIPVL
AEPHKRGTFTAIGLASRYLHTQGISPDEIVCFMPADPFVDAAFFEKVASLPSVLAQSGARMALIGTKPSFPSTQYGYIVP
ASAQDREAAPYLSVAEFAEKPDEQGASALLSRGALWNSGVFAFANSFMLEVLASRGLPDDYGQLLGLYDKLPELSFDEEV
VEKTKPIVVLPYDGRWDDLGNWQALAPHLEGRAVGPGEVSDDSADTHIVNELSSPVHVIGVSGIIVAASPDGILVASKAG
SSRIKQINPGKRRAPMYEEYRWGTVTVLDRVPPTGDANAGTEVVIRRITLLPGKSFPYRAHGSAKKSWTLLSGSGQMILD
GRITRVKTGDVLHLEPAAGYALLADTTGPVELIEIEFGPESAKPAPAAPSLSWEEAVLLCRNDREPDPGKG
>lcl|NZ_CP026520.1_prot_WP_042227190.1_3958 [locus_tag=PC41400_RS20010] [protein=alpha/beta hydrolase] [protein_id=WP_042227190.1] [location=complement(4462096..4462953)] [gbkey=CDS]
MEHGKEHSSAKRNYFRSGANELSYVDYGGDDRPVLLLLHGHMNDAGTFAFLASRLRKDWRVIGLDQRGHGWSGHPADGDY
SRESYITDILNLVRQELGGQPVVILGHSLGGVNAYQLAARYPEYVRAVIVEDIGAEIHADLSFAEKLPERSTTLQDLRDS
LARLGLKAIDYFSESVFEDEKGWGFRTDLTGMRSSQENINGQWWEDWLSSTCPILLIHGKKSFVMNAEQAKLMQSRRPNT
SLEIFENCGHGVHSDDPDGFYAVVKKFLDGLSESKIGNENVRGND
>lcl|NZ_CP026520.1_prot_PC41400_RS20015_3959 [locus_tag=PC41400_RS20015] [protein=YafY family protein] [pseudo=true] [location=4463086..4463974] [gbkey=CDS]
MSKSKRLMELMLTVNRMRKFTVGELAREFGVSQRTILRDLQELGELGVPLYSEVGPHGGYQVLRERLLPPIAFTEEEAVS
IFFAVHALRHYSSLPFETGTD*SDEKPGGFYHTRAEGCLSLFIHSARCGRSAEGAAHPVSIPGKKFRPRNPAHRHFYGKR
PLVLLRLLLSVERDACVPVRPHSFGCPFGSGTAGSPPYSS*KQGVCGTQGAGRDRCICGIDPRRGRGVRGGTLVGIDAYP
A*RWHRLASRNHFQKRSSLFREIFYRLGP*SNRERLA*ASGGNESHACRTARQIQL
>lcl|NZ_CP026520.1_prot_WP_009672673.1_3960 [locus_tag=PC41400_RS20020] [protein=hypothetical protein] [protein_id=WP_009672673.1] [location=complement(4464067..4464288)] [gbkey=CDS]
MEHYQTLIGVVLEKLSQSHQELKFNLDGLNSVLANHTAEEIAAAPELLTIRELRDVYADIVAAAEKRFPGLKG
>lcl|NZ_CP026520.1_prot_WP_042227193.1_3961 [locus_tag=PC41400_RS20025] [protein=FGGY family carbohydrate kinase] [protein_id=WP_042227193.1] [location=complement(4464306..4465823)] [gbkey=CDS]
MTYIASFDIGTTHVKGILVNRDGELFHEQNVSLTCRQEPPFIEQEPEQWFKAVGTIVNEWFRDGIGAGLISLVSFSGQMQ
DCIPVDEQGRAVRPAILYSDSRAGAEARMIRDEIGEEEIFRSAANRMDGTLTFPKLLWLKTHEADSCARTSRYLISSKDY
VIARLTGAFVTDPTSAATTGCMDIRRREWREDWLVRMGLPAGKLPDILPSGETAGRVHAEGSRLTGLAEGTPVLCGIGDA
GAATLGAGVYEPGEVYAYIGTTGWIAGVSGGVADPASGLFNLAYAEPGRYINVAPLTNAGNAHHWAAEVFAPEAAGTDGR
GMYEAVERLVEASDRGRNKILFLPYLNGERCPVQDEEASGCFIGLRPSTTKADMAAAVLEGVAFAMKQVMELRGAPEGAA
SLTLIGGGARSRQWCRVLADVFGTEVSVPSESQFLPCLGAAALGFPGLGWASSYDRLCELFKIRRKCETFSPDAEKKAHY
ESKYARYLELYPALAPVFRQAATEA
>lcl|NZ_CP026520.1_prot_WP_042227195.1_3962 [locus_tag=PC41400_RS20030] [protein=GIY-YIG nuclease family protein] [protein_id=WP_042227195.1] [location=complement(4465959..4466315)] [gbkey=CDS]
MDRRAELVRQFKEMETEAGVYQIRNTTNDKIFVESSPNFKNINGRTFELNIGSHLNKELQKDWTELGEEKFVFEILEIVE
KKKTGYFDLKDALKKMERKWLDRLQPYGDKGYNKPPRA
>lcl|NZ_CP026520.1_prot_WP_042227198.1_3963 [locus_tag=PC41400_RS20035] [protein=sensor domain-containing diguanylate cyclase] [protein_id=WP_042227198.1] [location=4466553..4468205] [gbkey=CDS]
MSLSLRTTFALVSALVICLLTGVLSTVIGKKTTRQVEESIGISLSQVAYQMADKLDHFMWSRAGEIDVLSELISNREAPG
PGDANELRSLLEQLKKSFPSYTWVAFIDPEGKVTASTDGILTGTNIASRPVFTEGKKGRFIGDVHEALLLAKLLPNPSGE
PLQFVDISTPVYNKQGQLTGVLAAHLSWEWSREVEATLLQPLKDKMKQLEVLIVSKKENTVLLGPKESLYRPLDTAGIRQ
AQAGGNGWLTETYPDGHTYLSGYAYGNGYLDYPGLGWSVLVREPADVAFTSVRELKTYITTAGIILSLVFAVLAAFLAGW
VVRPLQKITRSADRLRKGESASFPAFTFIREIRILSDSLRNLVDNLTKTEHALDQMSDLALHDKLTGLPNRIALDAYLEQ
AMRQAKENGTTLTFLYLDLDGFKKVNDSLGHLFGDKLLQQTALRLTQGMSTDSIITRVGGDEFVIVLPTGASAPDPAAIR
LAGSILERMNEPFLVDHQALRIGCSIGAAVWPLHHEDPIETMRLADQALYMSKRSGKNRMTFSPDSRENA
>lcl|NZ_CP026520.1_prot_WP_084706524.1_3964 [locus_tag=PC41400_RS20040] [protein=M15 family metallopeptidase] [protein_id=WP_084706524.1] [location=complement(4468265..4468843)] [gbkey=CDS]
MSAKKRLFRAAFAVMAAVLSAIMLIQLVMTKSDEKNLEKLGLDSARLLSGLDPGVARKMDELLAKTAAIGIRIVITDDFR
SMEEQDKLYSQGRTSEGKIVTHAKAGESYHNYGLAIDFALKTKSGKIIWDMNADLNGDGRKDWMQVVTIAKELGFEWGGD
WTNFPDYPHLQLSYGLSIAELQRGTKTARSGP
>lcl|NZ_CP026520.1_prot_WP_042227199.1_3965 [locus_tag=PC41400_RS20045] [protein=glycosyl hydrolase family 18 protein] [protein_id=WP_042227199.1] [location=complement(4468882..4470501)] [gbkey=CDS]
MNNKKWVAAAVLGCMLTGSATANAYESRLKPFGDVPESAWSAEHIYTLAAVQALDGYQDGGFHPLAAMSREEFIKLLVTA
AKTGYDAGHPVTLTDVGRDRWSYPMIGAALQRGWIKAMVDSGGAFHPAAPIRREEVAYAMGSLLLDKLSEAERASWLASG
WKREQSSRAFADAAGVDEALKPYVYYAVNRGMLEGSENGLEPDKQLSRQEAAAVINRLLHKEAGNKPLEVTGFYAVQSYN
SLGKMPLLDRVALGWWHLNYEGAGSAKLGSAGDPFKFNRPDGWEEVTASASSSHVSKDLMVFANKSHRVEEFLGDAAARK
AFTAALAETLSDPKYGFTGVCIDFEDLYGAGKKAEFAAFLSEVKEAAKGKTLTVTVPPTYYFEGYDLPSIGKTADTVILM
AYDFLYKPDRLPSAPLPLVAEAVRDTIKAGVPAEKLVLGMSKKTDQFIDQGGTTTYASPASSAVEERLKKPGVKSTLSVP
YLLGHIRYTDGGAHELFYEDSDSLDRKIWLARYYGLKGVSLWYMGQFTDSDWQRIDAVK
>lcl|NZ_CP026520.1_prot_WP_042227201.1_3966 [locus_tag=PC41400_RS20050] [protein=hypothetical protein] [protein_id=WP_042227201.1] [location=complement(4470625..4471527)] [gbkey=CDS]
MEKKRCLFCDQIVPIGREDGYDLYTGCYCSPDGSYGLHRDSYDPYSVLSYQTKHLMFPVISAYIREITDCGETVRLTIDD
LEGIRNSPRVPVTIEDKGIRLLQFFYRHSGGPDEAVVLQQLPKSYNLTYSPNLQELIYIIEKLKEERLLERMGSSFKLTA
QGWKTAEESLGGKRLKPCFVLLPEDVETRLKWTEKVLPVIEQCGYAPRTSEKIPREKEADYNIRMIADSKLLIADLGAQG
PEVFLAGGYALGRNIPVIWTARRAGDETTASPSEKIRPFVWETAEELASILKQRLTSEVF
>lcl|NZ_CP026520.1_prot_WP_042227202.1_3967 [locus_tag=PC41400_RS20055] [protein=hypothetical protein] [protein_id=WP_042227202.1] [location=4471728..4471943] [gbkey=CDS]
MAQNRRLTTDQDFKEAMDRQSPVRVFKNDHIIDTGAVITRFDDSMVVVQTRVSELTYHDRLGCEFFEMRKR
>lcl|NZ_CP026520.1_prot_WP_241688706.1_3968 [locus_tag=PC41400_RS20060] [protein=hypothetical protein] [protein_id=WP_241688706.1] [location=complement(4471986..4472453)] [gbkey=CDS]
MERREEPVILFQASLSLVVSAANKSQAAETAAFLLNRESIDLSPVQMVNGQGEKAEFRMESVDAVEWTRVEDIREGGRFK
VYGTIRLKLRAGSPENYASVIRAGLSGYHLPRSVIHDHTVWVIPTNCGPAFACVLDEKTSWKPAVQEPAMLVAVG
>lcl|NZ_CP026520.1_prot_WP_042227206.1_3969 [locus_tag=PC41400_RS20065] [protein=hypothetical protein] [protein_id=WP_042227206.1] [location=complement(4472717..4472938)] [gbkey=CDS]
MMSNNEEARKADFSNETPEITPKTPPEIKPNDVPEVAPRKVPETPPAAPPEIKPTVPPEISPQRNPEFPDPNR
>lcl|NZ_CP026520.1_prot_WP_042227208.1_3970 [locus_tag=PC41400_RS20070] [protein=TIGR02206 family membrane protein] [protein_id=WP_042227208.1] [location=4473077..4473817] [gbkey=CDS]
MERFFGANEAEGFTAFSGAHWAALTVFAALVLALYGSRNWFTRGNRALRGRIAIALILVLCETGLNVWYAAEGIYDVRYT
LPLELCSISLYLSVFMLLFKSRLLFQVVYFTGIGGALMALATPVLWYAFPHFRFFEFFAAHIAIILGVLYLIWVEGFRPT
FKSVFLAMGYLNVLLVAVGTANFLTGGNYMFLAHKPETASLLDYLGPYPWYLLSLEAAALVIFLLMYLPFALGSARPKNP
RLPHVR
>lcl|NZ_CP026520.1_prot_WP_042227210.1_3971 [locus_tag=PC41400_RS20075] [protein=hypothetical protein] [protein_id=WP_042227210.1] [location=complement(4474004..4474477)] [gbkey=CDS]
MKNRLIEDRVARLLHESNSQVQVTLESHFPGGRMAGGKYNMGRHLITLYIEEIKAQCMQLYGSHDLLEAYVDIVLAHEIG
HAEDGDLDRLASLMDTSDSELERRRIALRIEENAWAYARALLPETSGPMLDEIVRESLQAYHEAIDELELELQPIGA
>lcl|NZ_CP026520.1_prot_WP_042227212.1_3972 [locus_tag=PC41400_RS20080] [protein=acyl-CoA thioesterase] [protein_id=WP_042227212.1] [location=complement(4474639..4475151)] [gbkey=CDS]
MESKFVKETRCMKVARVFPPDVNNHNTMFGGKLMSYIDDIASISASKLCRVTAVTASTDSVDFLYPIRPSDSVSLESFVT
WTGRTSMEVFVKVITEDLVTGERKIAATSFLTFVGLDENNKAVPVPRVIPETEEENKLYETAEHRAEIRKVRREASKTFA
DFLLTKYPWE
>lcl|NZ_CP026520.1_prot_WP_241688707.1_3973 [locus_tag=PC41400_RS20085] [protein=response regulator] [protein_id=WP_241688707.1] [location=complement(4475396..4476082)] [gbkey=CDS]
MIHLHGMGKQMPYNQAEGGRSSGGADGSSAGPRSVKSKPRGYTFTSERKIRLPEKIAVSKELDLTGAYLLFRFSDSAGPV
YPALSLLPSASSRSDSRNGEDLSILIAEDNPMNRKLLQTILRKRGFEPDIAEDGAEAVEAFARKAYDLIFMDIAMPNMDG
LEAARRIREAAPGDNSPVIVAVTAYARKEDRMKCLASGMQDYVSKPFRIAEIEDVLMKWIGYIKEYKN
>lcl|NZ_CP026520.1_prot_WP_042227214.1_3974 [locus_tag=PC41400_RS20090] [protein=Na-translocating system protein MpsC family protein] [protein_id=WP_042227214.1] [location=complement(4476208..4476888)] [gbkey=CDS]
MEKKTALSQLSSQTGKMLRERFGKGPESIFISYNEHGITLHFRNFVSAVEMILLDKQKGEAVRETRQMVMDVLLPDLKEF
ITQTVGIAIEEIYYDWDLEDHSGIMIGLTNKSLDADIQEKYAGYEKVHQQVSEVTAHVQKIPDHIYSFWTNPRTLIIVRE
GLLIMLEKELINTGYENALRVAKTSLEKKLLLHDVDMEQAIGRTVTGLYLDWDFERDRSVLVYTFK
>lcl|NZ_CP026520.1_prot_WP_042227216.1_3975 [locus_tag=PC41400_RS20095] [protein=hypothetical protein] [protein_id=WP_042227216.1] [location=complement(4477242..4477580)] [gbkey=CDS]
MANEIQPVTDWLNSNANKAIRIAKEEQEDIDRVDLQLSRVEYREYQPDAPDDYTNGSALLLYGEGTVLNNGSEEALPQGT
FVIPLKGLSVADLNEDSVVLKTERAVYSLTLQ
>lcl|NZ_CP026520.1_prot_WP_042227217.1_3976 [locus_tag=PC41400_RS20100] [protein=thioredoxin family protein] [protein_id=WP_042227217.1] [location=complement(4477690..4478010)] [gbkey=CDS]
MSLTTVNDATYLDHISDGGVTVIDFTAVWCPPCKTLLPILEELSTEYAGKVNILKVDCDDSPEAASAFGVMSMPTVLVLH
RGEPVEKLVGLRPKSVYQSVLSRYIG
>lcl|NZ_CP026520.1_prot_WP_042227218.1_3977 [locus_tag=PC41400_RS20105] [protein=MerR family transcriptional regulator] [protein_id=WP_042227218.1] [location=complement(4478007..4478378)] [gbkey=CDS]
MKIGELSARTGVSIRSLRYYEEQGLLAPVRGDNGYREYTSFAVEQVRTIQLYLGLGLNTEQISGFLHCVLMNNEAFCSEV
LPVYKQKLSEIEEQIGVLQSIKSNLEERIHYILNNGTKEEKFQ
>lcl|NZ_CP026520.1_prot_WP_042227219.1_3978 [locus_tag=PC41400_RS20110] [protein=metalloregulator ArsR/SmtB family transcription factor] [protein_id=WP_042227219.1] [location=complement(4478556..4478885)] [gbkey=CDS]
MEQLQSLADTFKLMGDKTRLTILALLRERELCVCDLVDVTGMSQPSISQHLRKMKDAGLVSERRKGQWIYYSLAADHKDL
TLELLRQVPSMSHKIAELEENTDGSGRCG
>lcl|NZ_CP026520.1_prot_WP_009672917.1_3979 [locus_tag=PC41400_RS20115] [protein=YqkE family protein] [protein_id=WP_009672917.1] [location=complement(4479039..4479299)] [gbkey=CDS]
MGKKNRPAPAPRAADTDKPATLKDLLNPQVIEKLKAQSDQMKADEAKRREEEQRKAAEARKAEQKRLDNNFEHLLANSEM
DWRKYK
>lcl|NZ_CP026520.1_prot_PC41400_RS20120_3980 [locus_tag=PC41400_RS20120] [protein=SUMF1/EgtB/PvdO family nonheme iron enzyme] [pseudo=true] [partial=5'] [location=complement(4479391..>4479560)] [gbkey=CDS]
PSDNP*GPASGDSRVMRGGSYLCHKSTATGTGQPQGIKARRTAPPGISGSGVRPML
>lcl|NZ_CP026520.1_prot_WP_042227221.1_3981 [locus_tag=PC41400_RS20125] [protein=spore germination protein] [protein_id=WP_042227221.1] [location=4479698..4481200] [gbkey=CDS]
MPDFTIRSFNDFKTYCSQGTNQIQSKIIGIPGESYELFYLHQMTNKDILHRDILHPLLELGGGFGPDELLSYLPVGAGKK
ICEWGELPSLLLAGWVYVHPTGFERGVILQADQIPARSLTSAENESQIFGPQVSFAESIDQNMALIRSIINDPLLKTELF
TIGTQTKTRTIMCYMEGLNNNAYAEMVRKRLSNLKTDGILDSGMLAKLLEDNPDSIFPMYFLTERPDRTVGSLLEGKVIV
LVGGSPRVLIGPSTFIDFFKAEEDRYVHWQMSAFVFTLRMLAVFFSMLLTPLYVAALTFHYEMIPSALLDPLSSSRAKVP
FPPLFEALLLEFIIELLREAGARLPTKVGQTMGIVGGIVIGQAAVQAGFTSNLLIMIVALGALASFTSPGYMMGTAVRIL
RFPMIMIAGFWGGVGIVTAASFLLIHLLRLTSLGTPYLKPFYPFDNSEFTELFIRRPYGKFMDKLVKTFSFQARKKKGVN
SETPDNRQNIDLFEQFQEEN
>lcl|NZ_CP026520.1_prot_WP_042227222.1_3982 [locus_tag=PC41400_RS20130] [protein=GerAB/ArcD/ProY family transporter] [protein_id=WP_042227222.1] [location=4481210..4482325] [gbkey=CDS]
MNSYPESPHISLRAYLLFFIITTAQVGVGGASFQRKIYKMAGQDSWVTVLIAGLFCHLVIVAMFSFLKKYPGKDLYGIQK
EAFGKWLGGVMSFIYAAYLILIALTILRNYAEIVQTWIFPDISPWFLALMVLALALYGVLGGIRVITGIGLVSVLIITLE
MLLFYYPLQYANWGYLLPVMNHSPLELLKGTEEISFSVVGFELLMFVFPFIRDTNKAPKYAHFAILFTTGTYLIYMFTAL
VYFSGGQLIHTIWPTLTLLKIVEFPFIERVEYITVSLYLIGILPNVMLYIWSASRGIPKMSPLKEKAALYLVVGFLFVIC
TFFSSRLEIKRLNEIAGQISIYMAFVYPLLLSMLLFFKRRVSQPRNGGDSR
>lcl|NZ_CP026520.1_prot_WP_042227223.1_3983 [locus_tag=PC41400_RS20135] [protein=Ger(x)C family spore germination protein] [protein_id=WP_042227223.1] [location=4482322..4483455] [gbkey=CDS]
MKIRRLCLLALVTVLAATGCVERRIFERLGLVVALGYDKGQGSILSGTSVMYTLDKESGHRATILANQANTTKKLREDQN
RKSSKKLMLGQLRVVLYEEAIARDGLIDLIDSLIRDPSIGTKIYLGVSTEKSADLLQYPYRDIRNIGTYLYDTIKQNVLE
GQIISPTLHEFLRDYYSAGKDPVLPVFVRKGETIATAKTALFQSDKMVGMLSSSDAAYLKLIRNRFKAGTFELKLDKKKL
EPYLNENNADRREHIHVVVRIVGNKQKIRLTDREKPAFDLNMKLKFALEEISAEINLTEPQVTRTIEKLVTDEFTKKMES
VLAKMTDLKTDPVGFGEIYRQSLRGGSLTKSDWYDKMSSMSVDVHLNMELLRTGVVE
>lcl|NZ_CP026520.1_prot_WP_042227224.1_3984 [locus_tag=PC41400_RS20140] [protein=cell wall-binding repeat-containing protein] [protein_id=WP_042227224.1] [location=complement(4483570..4484943)] [gbkey=CDS]
MKKWIGVGLITLTASLALTGCLNDPSKNTAGGAAATQPATAAGSGEADSKAPWIASKNTTRINSSDPYEAAVLTSHTLWM
ATSDDNRPGGVVLVNPEEWQTALASADLIHHPNNGPVLFVNKDGIPDVTVRELKRLQPAGVESNQGVQAILVGNLDPKVE
KQAKELGLKTDKIPADNPAAAAKAIDAYYAKVAKENPPSVIVGSSESKEYTMPAVNWISHMPEPLLYVSKDKVPQETIDA
LKTRDGKANIYLLGPDSVISAAVEKQLGEYGKVVRIAGKDPVENSIAFAKYKDPATGFGWGITTPGHNFSLISKENPDLA
LASAPFSHLGKHAPLIWTEKDKLPDSVMAYLMAVQPKYDKSPVEGPFNHAWLTGDERSISASGQGEIDSMLEIVSKTGGG
HGGHGGMSGMPAAGDQSGTAGGDAGKNSGTGGTAGTGSGSASGPGKTDMSGMSHAGH
>lcl|NZ_CP026520.1_prot_WP_042227225.1_3985 [locus_tag=PC41400_RS20145] [protein=ATP-binding protein] [protein_id=WP_042227225.1] [location=complement(4485059..4486189)] [gbkey=CDS]
MKKKLHVRLAVMIVAMATGILIISTATILISTHYHITMYMNQADGAMHDMPRLNTHLEQAIMQSVLWTFLGSIMLAACMG
YFVARRVSAPLVEMKAVAEKMTGGQLDSRTAVGGDDELAELGRSINELAEQLQRLEKIRVTMTEDIAHELRTPLATLKSH
MRALEDGIWEPTPERIHSCYEEIVRLADLINELEELNELDSPVFRLTRTKVPLDTVMEKVKAFMSAAFLEKRIRFTVDAP
RNVYLDADPDRLTQILVNLLGNALTYTSPDGEVALKGRRAGDKVLITVKDTGPGIRENDLPYIFERFYRGDKSRNRRSGG
SGLGLAIVSKLVQAHGGRISAANEGGAVFLIVLSAAQPPPAAIKTQILRETDSGDE
>lcl|NZ_CP026520.1_prot_WP_042227227.1_3986 [locus_tag=PC41400_RS20150] [protein=response regulator transcription factor] [protein_id=WP_042227227.1] [location=complement(4486186..4486878)] [gbkey=CDS]
MKNILVVDDEDKIREVIVSYLQKEGFRTAEAGSGQEALRTVRQGGVDLLVLDLMLPDMDGEQVCREIRRSDAVPVLMLTA
KVSKTNQIHGLSLGADDYLTKPFDPRELVARVRAVLRRSDDSQLLADRVSFRNGHLEIDSLRQVVFCQGERASLTPIEYK
LLIALARYPKRSFTRDELVDKIMGFAYDGDVRTIDQHVKNIRQKIEKDPKNPVYIQTVYGAGYRFSGELK
>lcl|NZ_CP026520.1_prot_WP_042227228.1_3987 [locus_tag=PC41400_RS20155] [protein=hypothetical protein] [protein_id=WP_042227228.1] [location=4487265..4487630] [gbkey=CDS]
MIKKFSAILLTGVLAMSLATSAYAANESLQVISEPGEDNTAGSISGQDKEGKFTFFTTQSGGLNYAKAAVLSSCGGAPID
LEIKGVGSITKAIKLSSSCTYKVRLWNETGGTAKAFLRNWE
>lcl|NZ_CP026520.1_prot_WP_042227229.1_3988 [locus_tag=PC41400_RS20160] [protein=hypothetical protein] [protein_id=WP_042227229.1] [location=complement(4487871..4488893)] [gbkey=CDS]
MKEYLVKQAAICTLLAVSLLAGCGAREQNVQVKEPVQPDSARTASPSPGAAASDNVLAKLQLKIAQAKEAREVTAFLDEN
MPKADKATADQMFIELQKFYDTHLPGLNAGFTEMMGQPGTADKMREIGYPADFNKIVNDDTLKQWLKNQAAAKLQLSDKE
GSYFWKVDYDALNKSYAPYVSDDLKAYLAIQSLENKTPFRSDASLKISRDELGERILAAEKALTAYPKGLKAEEIMALYS
EYLHEYIHGYRYEAIEEPGMKLIPLVKRSYEKLVKNNPDTKTAGIVKEYLTVINANRDVIYSPGKKGESIIGDPKPGIDA
FWKGLDARIAAAFPDVKTAK
>lcl|NZ_CP026520.1_prot_WP_241688708.1_3989 [locus_tag=PC41400_RS20165] [protein=hypothetical protein] [protein_id=WP_241688708.1] [location=complement(4488934..4489932)] [gbkey=CDS]
MMFLFAAAPGCDRSPSPSVTVVDPPAAGTGVNTVLAGIQRLDQFYGMGWSPEGELIGISGASLAGAPLEFRNLAGDKIRD
TGIKGATGLQMAPDGKNAWVYHRYEGTADFYNLESGRITRTGPQSRELSGAWVDDKSYLAVLAEQKSRLVRFGIDGEMTP
VEHENVESGIVKAVFKAGRLYVLSGERELTVTETGGGGARAWTRSGVSDFALSPDGRQIALVSGTEPDQSVLVLADAVRG
GSEKTAARGRLLGQLSWSPDGSRVAFSVFSPDQGMTGLYVMNADSGRMIPVSYKPNLKSVIDWSPAGTSFMVSEVLSGDI
TDNMFTTIYQLK
>lcl|NZ_CP026520.1_prot_WP_042227233.1_3990 [locus_tag=PC41400_RS20170] [protein=HAMP domain-containing sensor histidine kinase] [protein_id=WP_042227233.1] [location=complement(4490078..4491478)] [gbkey=CDS]
MRISLKWKFALFIAALLLFAVSVLSVLVLNGVHRYQKEEMEKTLDRQAEIANNWIGQQVVTGTRFDPQTFMGLQGQNLAV
EIGAASGMRFLLYDMTGARTGDSLPLAESPDVSGVLSHALNNKIVYRVLGDTEDYLAPAQGPDGQIGVLHFQVSLAASNA
FYRNLQNSFLIAGSGVLLFSFVLGLLYMNRQAKAIRRLKEAADDIRVGAFIDAPPFRRQDELGSLSDGIYEMSRAIQGSL
RTQQHFINNISHELKTPLTSIRAYTDLLHMYEDDPGLVGEARDVIDKEAERLYELVEKVLKLAALEKYEFEQQLEKIRID
SLLEDLTSRMRGKAEKFGVTISHALVPAELWADRESLIQIFLNLLDNAIKYNVENGTVLIRSRIMEEGGVEVEFTDTGLG
IPEDRRERIFEPFYTVNKDRARKSGGSGLGLSLVRQLVDKQHGQIRVEHPVKGGASFIVTFPAGDE
>lcl|NZ_CP026520.1_prot_WP_042227234.1_3991 [locus_tag=PC41400_RS20175] [protein=response regulator transcription factor] [protein_id=WP_042227234.1] [location=complement(4491469..4492164)] [gbkey=CDS]
MDYKILVADDEQTITRAIAYGLKREGFVVETAADGEEALRKTESFRPHVLVMDVMMPRLTGFEVCRKLEDRKNLGILLLT
VKNDIVDKIVGLEMGADDYMTKPFDLRELIARVRALCRRLDKTGNGEESAVLTIGELSLHPEQRLVSLRGTALNLTPKEF
DLLALLLSNLKRAFSREELLDLVWGMEYAGGTRTVDIHVQRLRTKLGEPYQHVLQTVYGIGYKAIGEVPCG
>lcl|NZ_CP026520.1_prot_WP_164977146.1_3992 [locus_tag=PC41400_RS29915] [protein=hypothetical protein] [protein_id=WP_164977146.1] [location=complement(4492429..4492602)] [gbkey=CDS]
MSDKHPKGRVEFRQPPVKLTADLPGDPEMREHLDEWRENDGYPGSCGIINPPEGQEK
>lcl|NZ_CP026520.1_prot_WP_042227236.1_3993 [locus_tag=PC41400_RS20180] [protein=nitroreductase] [protein_id=WP_042227236.1] [location=4492927..4493529] [gbkey=CDS]
MPTDMNLTEAISGRRSIGRVKDEPIPREIIEELLEAANWAPSHRCTEPWRFFVMTGAGRGVLAEAYADVAAEGSGASGEA
LEELRAKQGSKAYRAPVIIGVAVSPSEDPAVNRMEEFAAAHSAVQNMLLAAHGLGLAAIWRSGDPMYHPRMKAAFGLREN
DEMVALVYVGYPQIAPQAGKRTPVQEKTVWITEGSGPSAE
>lcl|NZ_CP026520.1_prot_WP_042227237.1_3994 [locus_tag=PC41400_RS20185] [protein=PadR family transcriptional regulator] [protein_id=WP_042227237.1] [location=4494036..4494389] [gbkey=CDS]
MDILDTEKVLKKYVPMTETAFYILLSLAEPRHGYGIVKHVEEISNSRIRLGSGTVYGTLTKMQKDGIITVFSDEERKTVY
EVTEIGHKLITAEIDRLKELYQNALKYGEGFYDNKSI
>lcl|NZ_CP026520.1_prot_WP_042227238.1_3995 [locus_tag=PC41400_RS20190] [protein=DUF2812 domain-containing protein] [protein_id=WP_042227238.1] [location=4494370..4495569] [gbkey=CDS]
MITKVFRPFWSYDVQKTEKWLSSMAEKGHELVKINRGTRHFFFQQGEPRKRTYRIGFNKIQTNSLSRVLLDDGWVKVLQS
GNWYVTANEQPLEQIKTFPVREDIVKHNKTILYIFASVLIYLTVISIFNLILGSVIFFQSVPGHFVESPLWILTYFLIGI
GIALWVLTLYSVIKINKTNKKLIGENIQRKELQGREQDERRLSKDEEKRLRRSGQLIVKRRFGWFYAPDKLEKWLEAMEE
QGFNLYRVGKTGTVFYFTMGSPRKVSYCADYQNIADEGYFDIHRDAGWKSAFISASSIQKWTLWSREYSMGEEPPQIYSD
KSHQLKHARRIAITYSCMFLPLVIFYIYIIGANIHLMSNHNLDKMQIINMTLFILLILNFGSFSIRTWLYYRRLSKRYK
>lcl|NZ_CP026520.1_prot_WP_042227239.1_3996 [locus_tag=PC41400_RS20195] [protein=MerR family transcriptional regulator] [protein_id=WP_042227239.1] [location=complement(4495835..4496623)] [gbkey=CDS]
MAYKVKEVTDLVGVSVRTLHHYDEIGLLVPDSVNAAGYRLYTDRNLERLQQILFFKEIGFALQEIKAMLESPDFDRKRAL
IGHKELLLRKKKRLEDIIETVDRTIASVEGGIPMRKEQLFESFDRTPLEEHKKKYAAEAREKYGSETMDAVEKRAGSYTK
EQWADIMGQSERIYAKAAAAMEAGPSSRMAQEAAAELRQWITGHFYDCTPEIFRGLGDLYVTDERFTANIDRHGAGLAAF
LKEAMHIYCDRLEGAGEDTANR
>lcl|NZ_CP026520.1_prot_WP_042227474.1_3997 [locus_tag=PC41400_RS20200] [protein=DEAD/DEAH box helicase] [protein_id=WP_042227474.1] [location=complement(4496669..4500019)] [gbkey=CDS]
MSSALTRKAIQLLCGRTAYDKGEALNRSGKVVFTYHDPVLQSVEATVDDRESYLVKVSVLSGGGINAECTCPSLPSFRQS
CKHTAAVLLKLLDSSRIDKDFPGAALLERSSEGRSESTAAPSGLPEARQAEEASPYDTELTSSMLGLFGGSTRPGSTRFT
RDERETLELEILIRPFAYGFRKVMFAAELKIGPKRVYIVQKIRDFLERLDRGEAYEFSRVFTYEPNVHGFRREHDEVLRQ
FVKIYQNENLYRETGYSSASSAKAFPGGERMLPIPPFSWTALQPLLAHAPGVKLELADGVYDGIRVSGEPLPLQFDFDRS
ETDGYRMDVQGLDRITVMETYGIVLHNGELLELPAHHCKRLSELKRLMETAHRNRLTITPGQMEPFMEKVVPGLMKLGKV
RIAPSVSDRIYQTPLKAKLYLDRVRDRLLAGLEFHYGSVVLNPLEEGGRQRGTDHIVMRDTEKEETIIRLMEQNSFAKTE
GGYFWDDEEAEYDFLYHTVPLLERMLDIYATSAVKERLYTGFPPQITVEVDERTDWLEFKFELLGLPEAEIRNLIRSMQI
KRKYHRLPNGSLVPLESAEFQEIIALMNEAGAGKGDMTDTGFRLSVNRGLHLTDARERGSSVKLGKSFRRLLENMRNPDN
LDFPVPERLSPILRDYQKFGYQWMKTLAHYHFGGILADDMGLGKTIQSIAFLVSVLPDIRKKKLPALIVSPASLVYNWLH
ELKKFAPEIRVVIADGTKTQRSRALRKGSKADVVLTSYPLLRRDKDLYAKESFHTLILDEAQMFKNYTTQTAQSVKQIKA
AYRFALTGTPVENKLEELWSIFDAVFPGLFPARQAFNELSREAVARRARPFLLRRLKTDVLKELPDKIETVQASELLPEQ
KKLYTAFLAKLRHETLKHLHEESFQQNRIRILAGLTRLRQICCHPALFVDGYEGSSAKFEQLLEIVEECRSAGKRMLIFS
QFTEMLGIISRELGYEGVPHFYLDGKTKGSERVELCSRFNEGEKEVFLISLKAGGTGLNLTGADTVILYDLWWNPAVEQQ
AADRAYRMGQKKVVQVIRLVSQGTVEEKMFQLQQKKRTLIEDVIRPGEESVSTMTEQEIRDILMLDEKMEIGEIGD
>lcl|NZ_CP026520.1_prot_WP_042227240.1_3998 [locus_tag=PC41400_RS20205] [protein=heavy metal translocating P-type ATPase] [protein_id=WP_042227240.1] [location=complement(4500321..4502768)] [gbkey=CDS]
MAANVKAAGEEATKQANLQIAGMTCAACANRIEKGLNKVEGVQAANVNFALERATVTYDPSRVNTAQLEEKIGKLGYSTV
KDFVDFQLEGMTCAACANRIEKGLAKMPGVTSATVNFALETAHVEYQPGEVTPADMQGKVEKLGYKALLKQEQGDPAVHR
QEELKRHKRKLILSAILSLPLLWSMVSHFSFMSSWVWLPDLFMNPWFQLALATPVQFYIGRQFYTGAYKALRNKSANMDV
LVSLGTSAAYFYSLYLTVEWARHGGSVHHGPEMYYETSAVLITLVLLGKLFESLAKGRTSEAIKSLMGLQAKTALVIRDG
EETAVPVEEVIAGDIVLVKPGEKIPVDGEVLEGSSSVDESMLTGESLPVGKKAGDPVIGATLNKNGSLRIRATKVGKDTA
LAQIIKVVEEAQGSKAPIQRVADVISGIFVPIVVGIALAAFVIWYFWVTPGNFAGALEKAIAILVIACPCALGLATPTSI
MAGSGRAAELGILFKGGEHLESTHGIDAIILDKTGTVTKGKPELTDVLTASMDEKRFLELLGAAEKNSEHPLAEAIVSGI
AARGVSVPASSEFEAIPGFGIRAVVDGQEVLAGTRKLISRYEVDAGHAGETMARLEKEGKTAMLVAVNRSYAGLVAVADT
VKETSKEAVARLKGMGIEVIMITGDNERTAQAIARQVGIDQVRAEVLPEGKAEEVKKLQAQGKKVAMVGDGINDAPALAQ
ADIGMAIGTGTDVAMEAADVTLMRGDLNSIPDAIYMSRRTMSNIRQNLFWALGYNTLGIPVAALGLLAPWIAGAAMALSS
VSVVLNALRLQRVKL
>lcl|NZ_CP026520.1_prot_WP_042227242.1_3999 [locus_tag=PC41400_RS20210] [protein=hypothetical protein] [protein_id=WP_042227242.1] [location=4503025..4504278] [gbkey=CDS]
MNTTNFSVKKTALLAVLTAVSLTSCSSIGPAATEDAAGSAAASAAASPGPASGTAAPASTAPVPSVKPDQIQTRLTDFQL
ISDAAGFAWGETNGSLRLYRTDDGGRNWTLLSPWDDTFKSGAPTAGGSLHFLDAQHGWTAAADDGRKTVTAFTTDGGRSW
TRGTLDKATYAAGLSFADPLHGWMLTSSDAAMGKSEKTLYATDNGGKAWKPLTSVSLDKPVPGALPLSGHASGMTFKDTG
NGWIALAGHEEKPVLYRTRDGGKTWNNVPMKLPQALSGSSVTLDGAPVFFKDEPGKGWIPVRTVSANTNSRDAYVTEDGG
QTWSFKQTGMQNALTFADSLHGWGFKDKVLHATSDGGATWNPVPSDPVLTDTLKNNPNVIRFEFAGPQTGWLLLGSEDGK
KSKLLKTTDAGKHWILL
>lcl|NZ_CP026520.1_prot_WP_042227475.1_4000 [locus_tag=PC41400_RS20215] [protein=AAA family ATPase] [protein_id=WP_042227475.1] [location=complement(4504370..4505089)] [gbkey=CDS]
MYLHEVIKLKENIRNPHEYPFHIPAIDKLDRLRFHKNVTFFVGENGSGKSTLMEAIAYQCGFNTAGGSRNNLYELDSSEA
KLGDYLRLSWMPKMNQGFFLRAESFYNFASHIDELAREDRRVYDSYGGRSLHEQSHGESFLSLFKNRFGRSGIYLLDEPE
AALSPARQLAFLSIIHELARTSQLIIATHSPILLGYPHADIYNFNESPVRLTTYEETDHYRITRRFLENRSFYLHELLD
>lcl|NZ_CP026520.1_prot_WP_042227247.1_4001 [locus_tag=PC41400_RS20220] [protein=(2Fe-2S) ferredoxin domain-containing protein] [protein_id=WP_042227247.1] [location=complement(4505423..4505806)] [gbkey=CDS]
MTTWNLNGMKHHLFFCNGGSCTKKGAEEVVEAMRDEIKLQEADVRIHTTRTRCQGRCEDACVITVYPEGVWYKEVTPEIG
RRIVREHLVGGSPVEEHISYTYEDELAARPDTKEGQSKTKRKEAQTK
>lcl|NZ_CP026520.1_prot_WP_042227248.1_4002 [locus_tag=PC41400_RS20225] [protein=DUF4083 family protein] [protein_id=WP_042227248.1] [location=complement(4505968..4506222)] [gbkey=CDS]
MDTIGFISWASVVYLLVIIGFAVIFVISLSMFIRTRVNRRLFTLTKLQRIEEKLDRLLDMQPQNAKMNPDRERIGNEPED
ERRY
>lcl|NZ_CP026520.1_prot_WP_042227250.1_4003 [locus_tag=PC41400_RS20230] [protein=VOC family protein] [protein_id=WP_042227250.1] [location=complement(4506270..4506656)] [gbkey=CDS]
MKKYIQHQAPVLPVINVFKALEFYRDVLKFDLAWVLEDTYGSVYNGNIEVHFTKSDKVEVQTLYFFVRNADQVYQFLKTQ
QVDFIEDIQSKSWGMREFVIRDLDGHRLRIGHGERNKHEIEVLSREIG
>lcl|NZ_CP026520.1_prot_WP_129112517.1_4004 [locus_tag=PC41400_RS20235] [protein=hypothetical protein] [protein_id=WP_129112517.1] [location=complement(4506670..4506861)] [gbkey=CDS]
MKKGTYFVFSCLLSLLSCFTHPEGFGLYTWWMYLVSNLVGLNLLVWIVGIIRERRTDRHLRSL
>lcl|NZ_CP026520.1_prot_WP_042227251.1_4005 [locus_tag=PC41400_RS20240] [protein=MFS transporter] [protein_id=WP_042227251.1] [location=complement(4506929..4507486)] [gbkey=CDS]
MNIRWLIRFQSVITLASGMIYPYYLLFLKNLGNSYSKYGLAFAVFTVSSAAVSRYLGPRLDRFGREILALSSLGMMASML
AFPWVGSYVWVLVLQAGMGICSAMQRMSERLLLADYTDCGARGPALGNYHFWTSVSSGFAVLAGGYLIDWLTIDVLFYLS
ALLYAASAWMVWRLRGGTSRKRAAG
>lcl|NZ_CP026520.1_prot_WP_241688709.1_4006 [locus_tag=PC41400_RS20245] [protein=FAD-binding oxidoreductase] [protein_id=WP_241688709.1] [location=complement(4507522..4509039)] [gbkey=CDS]
MPEEKKTRNENFKSGLPDPDARAPSNDPASPGKGFVSGSAVRAAAFVLLTVLCILAYIRTGTPDQDPFLITDYSRLHPVK
VERVVEGREEEQLVKLLEEARAKKLSVSVAGQRHSQGGHTFYKDGIVLDMTPFNRVLSFDPGARTIRVQAGATWKDVQDF
VHPYGLAVKTMQSQNIFTVGGSVSINAHGRDIRNGSMISSVESFRLLTADGVIKEVSRTENADLFPYVLGGYGLFGVILD
VTLTLAEDEVYRMTTEATRVEAFSTYFKERVKGSPDVRMSIARISVAPEHFLTDMYATLYTADPSLKLSEYNRLVTREYG
VLPSKVLFHLNRSSDWGKNLFWALQKNYFSNRQDEIVSRNNAMRSESAFMEYREAGKNDLLQEYFIPVGEFAGFVDRLRS
VLLEEDLNLLNITVRYVNRDEEAVLSYAQDDMLALVCLFNTSLSEEAQTRLKAGIRRIIDEVIRSGGTYYLPYAAYPSVE
QFRTVYRGTSNFSVKKTRWTPVICL
>lcl|NZ_CP026520.1_prot_WP_268630170.1_4007 [locus_tag=PC41400_RS30420] [protein=hypothetical protein] [protein_id=WP_268630170.1] [location=4509267..4509392] [gbkey=CDS]
MKLAIAVGYTAVQAFLGVNDVTIGEATLLSTEEILGSCGLF
>lcl|NZ_CP026520.1_prot_WP_042227253.1_4008 [gene=fabF] [locus_tag=PC41400_RS20250] [protein=beta-ketoacyl-ACP synthase II] [protein_id=WP_042227253.1] [location=complement(4509434..4510672)] [gbkey=CDS]
MERVVITGMGIISPLGNGVAPFWERLVQGQSGISRIDTFDTTRYKAKIAGLVRDFDGEALFGRKEARKMDRFCQFALAAA
DQALLDAELNLDMTDRERVGVYVGSGIGGIGTLLEQNRVLLERGPERVSPTLVPMMISNMAAAMISIRHQTLGPTMSPVT
ACSIGNSAIGEAFRTIRTGDADVIIAGGSEAAAGEISLASFGNATALSTCNEEPAKASRPFDAGRDGFVMAEGAGILILE
SLSHARRRNARIYAEVTGYGASSDAYHMVATHPEGRGAQQAMKSALASAGIAPSEVDVISAHATSTGVGDKSETIAIKNL
FGEDAYRIPVTANKSMTGHMFGAAGGAEAIALVKTLQEGIIPPTINLEQRDPDCDLDYVPGTARKAELEIGLSNSFGFGG
HNAVIVLRKYRE
>lcl|NZ_CP026520.1_prot_WP_042227255.1_4009 [locus_tag=PC41400_RS20255] [protein=helix-turn-helix transcriptional regulator] [protein_id=WP_042227255.1] [location=4510795..4511634] [gbkey=CDS]
MNKDERLEALSAFLKSQRAKLLPQNMGLPAGTRRRTPGLRREEVAQLAGVSTTWYTWLEQGRDIKVSASVLDNIASALQL
TVDERRYLTSLALETSSGIHLLQEELPQISPSLQKILQELKNCPTIISDRRCTIVGWNEAATRVFLDFEKIPPEQRNMIS
LLFARKEFRRLAVNWEDFVSGFLAIFRAYYGQYFEDEWYNGFLAEMKLTYPEFNRLWEQSKVSSAPEVLIEFRHAKAGKM
LFDLTSLQVRGSTDLRCSIYTPAQESSTEMKLMQLMDRS
>lcl|NZ_CP026520.1_prot_WP_042227257.1_4010 [locus_tag=PC41400_RS20260] [protein=hypothetical protein] [protein_id=WP_042227257.1] [location=complement(4511992..4512231)] [gbkey=CDS]
MTEPKKPSHELERQTQQKGNRDHPEQFPTENESLFDLHESEKNVDPIPLEDLNMEQKEEKDKTATKSDSSSPDKYHSGF
>lcl|NZ_CP026520.1_prot_WP_042227259.1_4011 [locus_tag=PC41400_RS20265] [protein=DNA topology modulation protein] [protein_id=WP_042227259.1] [location=complement(4512349..4512870)] [gbkey=CDS]
MKLLLIGPGGSGKSTFARELGEKLELPLYHLDAYYWKPGWVPTPNEEWDEFIGKLIREPAWIMDGNYGRTLEMRLKAADV
VIFFDLPSWLAVYRVIKRRIMYHNRTRPDLNEGCPEQLDWDFVKWVWNFRRHKRPQLLEQLENCGDNVRVFIVKNGKDKQ
DILEELLSLRSFV
>lcl|NZ_CP026520.1_prot_WP_042227260.1_4012 [locus_tag=PC41400_RS20270] [protein=YrdB family protein] [protein_id=WP_042227260.1] [location=4513065..4513403] [gbkey=CDS]
MAAVLAANLGIRFILECCLLVAVGNWGITKGDGWMMKTLLGAGAPLLTAVVWGLFMSPKASYPLPQPWHLLMEAAIFGLG
TASLFAAGHTRLGFLFLSSAVVNRLLMAVWSQ
>lcl|NZ_CP026520.1_prot_WP_042227261.1_4013 [locus_tag=PC41400_RS20275] [protein=spore germination protein] [protein_id=WP_042227261.1] [location=4513540..4515159] [gbkey=CDS]
MAFFLRGFTKIANRPPKQPRQQPSPGSSNAGSPEAACPAEEPCDASLAERTGWLKTQLAHCSDVVYHTFTAGTNTPCLLV
YIQGSVDRNDLQQSVLKVILGDSFVQGAEHLAGNLFDRKMIPADRQKTLLTMTDALQDILEGSAVLLLDQEPRMMSFGLS
SYIKRQIDEPNSETVIRGPREAFVESIQDNLSLLRKRLRTHNFKCVEQIIGTKTRTRVITVYMEDVCSPALVDEVARRLA
DIEIDSVMGSSYLEEFIEDNPYSPFPQLQYTERPDVVAAALLEGRITIIVEGTPIAIIAPTTLFMMMQSAEDYYQRFWAG
SWIRWIRYVFLFVSLLLPSFYIAITTFHPETIPERMLITIASSREVVPFSAFVEALIMEVFFEALREAATRIPKSVGQAV
SIIGALIIGTAAVQAGIASAAMVIIVSLTGIASFIIPHYDLGLSFRLLRFPIMILAGGFGLVGIACAMILIYIHLTELKS
FGIPYLSPVVPMVPKDMKDIFVRAPWWTMITRPKFTSSRNLNRQKPNGRKWENPAEEGD
>lcl|NZ_CP026520.1_prot_WP_042227262.1_4014 [locus_tag=PC41400_RS20280] [protein=Ger(x)C family spore germination protein] [protein_id=WP_042227262.1] [location=4515166..4516353] [gbkey=CDS]
MRMTARLAGILLSLFLLAGCWSSVELNDRAFVRIVMVDKTDDGYELVLGMPLANRLIPGEVGGGSQSGREPFTYFSKIET
NLSDAFRKIQNDLSRKITFGQTHVVIIGNRYAEEGIGSLLDLIAREPRVHINANLFVTHGSVDQIIHLPLTMERFPSDIF
YAYGKRKIATAINFKDILRAHYMGGDFLVTRLVFGKAGIMNESKEENWMGPDGAAVFKADKMVGSLPGKKAAGALWIQGQ
PETREITVPSPTDGKAVSLLVERNSTRIRPELTGDTFTYKIHVNTDLDIVASYSNINLKDHKQLKEMEKSVENKIKEDVE
HSIRLTQQMGTDAFQLNRFLEWKYPKEWNRIRKEWDEIFSKKVNIQCTSDATIRRIGVVKKTFTLDPQKDKEVSK
>lcl|NZ_CP026520.1_prot_WP_042227263.1_4015 [locus_tag=PC41400_RS20285] [protein=hypothetical protein] [protein_id=WP_042227263.1] [location=4516350..4516562] [gbkey=CDS]
MITVLLTYAIVMIIQWRYLVRCGKKKRTFSVSLSVTGVLCACVLLLRHSNNGESLIEWLEIVFGSLHVTS
>lcl|NZ_CP026520.1_prot_WP_042227264.1_4016 [locus_tag=PC41400_RS20290] [protein=GerAB/ArcD/ProY family transporter] [protein_id=WP_042227264.1] [location=4516583..4517692] [gbkey=CDS]
MDKITPLQTYMLFTQYLLSTLVSFLAAPLIQNAGYSVWIPVIAGCLLGGTLTYLAYKLSCRRPGGYMAEYGGEIVGRPLH
YLFILLMAFIYLFSAALVLRQLTDFIVEIYLPGTPLWAVAGLLGLCTTRLVHSGITNIFRSTQGVFFVTLALVVLIAVAS
CQDIVPYSVVSLFTNFDGRGILSATAMITGLFGEMTFIVLFFPKFKETGRTMKALAWAIVTAGILTLGIMIPMVLLFGPS
LAANLTYANLEMTRYIQLGNFIQNMDPILILMWLFTLLVKISLFLYISVITVTQTLKMKDTRLLTLPFGVTVIFLCLWMV
NSSETLDRFAGTTAVPMTLLAELIPVVYLLIDALRSRFKKRQPVREPST
>lcl|NZ_CP026520.1_prot_WP_042227265.1_4017 [locus_tag=PC41400_RS20295] [protein=N-acetylmuramoyl-L-alanine amidase] [protein_id=WP_042227265.1] [location=complement(4517803..4519080)] [gbkey=CDS]
MAVSQHPIIGNSLASLDVITAFIRKVNPSFNPEIARQFLTVGAKYGVRGDVAVAQSIHETDWFRFGGDVKPEQNNFAGIG
ATGGVPGNSFPTIKDGVTAQIQHLFAYASPNPLPAGEALVDPRFALVARGIAPNWEDLAGRWAVPGYDRSKYASLQAALE
AGETYGHSILKLYDGMTATAPVPTVKPLIVIDAGHGGTDSGASGSGLLEKNLTLTLALLVRDRLVNGYAANVKLTRSTDV
FVALSDRANLANGWGAAYFVSIHINAGGGEGFESYVYPGTSGGVTGQKRTIVHDTIVKYLSALKVVDRGKKEADFAVLRE
TGMSAVLLENLFIDNAADVSLLGDSGVLTNLANVIGDGVAKAMGLNPEIQPVIPAWKTEGVDWLYEKGLLTDPAWKDKLD
EPLPLWAEAVILKRLYDLLSGDGTD
>lcl|NZ_CP026520.1_prot_WP_174446788.1_4018 [locus_tag=PC41400_RS20300] [protein=nitroreductase family protein] [protein_id=WP_174446788.1] [location=complement(4519289..4519888)] [gbkey=CDS]
MMDFNTAVKNRRSIYAISKEAVVSDEKIVELVNEAVKHTPSSFNSQSARVVVLLNEQHDKFWDLTTEILRGIVGDGEKFE
ATAQKMGLFKNGYGTVLFFEDQAVVEGLQQQFAAYADNFPIWSHHSSGMLQFVVWTSLEAEGFGATLQHYNPLVDEAVRT
EWNVPATWKLVAQMPFGKPAAEAGEKQFQPVEDRVKVFK
>lcl|NZ_CP026520.1_prot_WP_042227268.1_4019 [locus_tag=PC41400_RS20305] [protein=multidrug effflux MFS transporter] [protein_id=WP_042227268.1] [location=complement(4520062..4521318)] [gbkey=CDS]
MSNNSLAAAEGAGRSRKFTLAVILGSLSAFGPLSLDMYLPALPKLADDLNTTASLAQLSLTSCLLGLAIGQVFAGSISDI
KGRRGPLMIGLAVYTAASLLCAIVPSVWGLIVLRFIQGMAGSAGIVLSRAVVRDMYSGPELTRFFSLLMLINGVAPILAP
IAGGQLLQFTSWQGVFIVLALIGVIMLLASWKGLPETLPADRRTKGGVKQTVATFKTLLRDRIFMGYALSQGLVAAAMFA
YISGSPFVIQNIFGASPQMFSLFFAINGLGIILAGQIAGRLAGRVSETKLLISGLIIAGLGGVTLLAMILAGAGLTAILI
PLFFVVSSVGIVNTAGFSLAMQNYGHSAGSASALLGLLSFIFGGLVAPLVGIAGSGTAVPMGIVIAATDVGAILCYYFMV
KRHISRTGQSAESAAGTR
>lcl|NZ_CP026520.1_prot_WP_042227270.1_4020 [locus_tag=PC41400_RS20310] [protein=lactonase family protein] [protein_id=WP_042227270.1] [location=complement(4521523..4522605)] [gbkey=CDS]
MSGQGERLLVFVGSYAEPEGPGVYVMELNEETGALTKLDEVSGLKNPTFLNLDVEGRKLYAISEGTTSEGARTGSAYAYT
IDSGQGSLTEMNHAVTVDAPTCHIQRDASGRYLVVASYHGGLVGLVEVRADGTVGRLLDVKQHEGHGPNPERQEQPHPHS
AFFSRDNRFVFVPDLGLDRIVSYVLDTENNVLKPHGETAVPPEAGPRHMAFHPGGKLAFVINEMSSTVTSLRYDEETGVL
EPIETVTTLPEGFTGESYCAEILVSEDGAYVYGSNRGHESIVVYSVDEASGKLTVVEHVSVEGKHPRNFALTPGGAFLIA
ANRDTDNIATFRVDRSSGRIQYTGQSAAVSKPVCVRPAYL
>lcl|NZ_CP026520.1_prot_WP_042227271.1_4021 [locus_tag=PC41400_RS20315] [protein=DUF421 domain-containing protein] [protein_id=WP_042227271.1] [location=complement(4522787..4523647)] [gbkey=CDS]
MPDWIVVVLRTLMVIVIVLAAAKLLGKRQNSRLTFFEVTTGVAAAVLAVYITLSRTADWYLGIIALFVWILAAVGIGFLQ
MKSKAARDFIDNRETILIKKGKIMEENLKKERLTTDDLLQSLRRKNVFQAADVEFAVMEASGEINVLLKKENQPLTPKHL
GIKVGPAEEPQAVIMDGKALDEPLAAIGLNRGWLQTELETMDATIDNVFLGQVDSYGELYVDLFDDKLQVPQPQKKAVLL
AQIKKCEADIEMFGLSTNDPEAKRMFEECSRSMEEMISEIKPLLSR
>lcl|NZ_CP026520.1_prot_WP_042227273.1_4022 [locus_tag=PC41400_RS20320] [protein=DUF1657 domain-containing protein] [protein_id=WP_042227273.1] [location=4523853..4524059] [gbkey=CDS]
MTVGSQVKTCLASLKSAQASLESFALSTQNQDAKTLFENAAKTTQDVVTQVQGRVQQLENEEPQYQGF
>lcl|NZ_CP026520.1_prot_WP_206735021.1_4023 [gene=cdaS] [locus_tag=PC41400_RS20325] [protein=sporulation-specific diadenylate cyclase CdaS] [protein_id=WP_206735021.1] [location=4524239..4524850] [gbkey=CDS]
MPYADCDFSPMKQNLKDDLEKLVLRLQRSIQTLDEENHCMLGEIEGIKENFLNIESRASSFYLNCYLSPFTDKYTELSVS
AQNLSKRRHGALIVVQRNDSLDGLIQPGVAIGASVTQSLLESLFYPGTPLHDGAVLIKFNEIVSAKNILPLSHIVQAEPK
LGTRHRAAIGITELSDAIVLVVSEETGRVSFASKGYLYPIVPR
>lcl|NZ_CP026520.1_prot_WP_042227277.1_4024 [locus_tag=PC41400_RS20330] [protein=DoxX family protein] [protein_id=WP_042227277.1] [location=complement(4525081..4525497)] [gbkey=CDS]
MMNGKTEIGLLIVRLVLGVTFLVHGLDKFQSGLGNIAGWFESIGLPGFLAYVTAVIETVGGAAIILGLGTRIAAALFGVL
MIGAMFSVKFSAGFTGNGQSAGFELDLALFALSLLLVLTGSRYLAVDSLFGRSRQTES
>lcl|NZ_CP026520.1_prot_WP_042227279.1_4025 [locus_tag=PC41400_RS20335] [protein=class III extradiol ring-cleavage dioxygenase] [protein_id=WP_042227279.1] [location=complement(4525568..4526335)] [gbkey=CDS]
MIPSLFLAHGSPALAIQDTEYTAFLRELGQQWKPKAIVIFTAHWESEVLTLTSTDETYETIYDFYGFPDELYRVAYPAKG
SVKLALRLEGLFRERGIEVDTDSARGLDHGSWTLLYKMFPEADIPVIQLSVNPHLPPDKQTAIGEALRGLGEEDILVIGS
GVTVHNLRMVNWGQEEPEPWAVEFDDWLIRHTLERDTAALYRYEALAPHSRMAVPRAEHFVPFFLALGSGSEERTPRVIY
RGYEFGTLSYLCLQF
>lcl|NZ_CP026520.1_prot_WP_042227281.1_4026 [locus_tag=PC41400_RS20340] [protein=winged helix-turn-helix transcriptional regulator] [protein_id=WP_042227281.1] [location=4526484..4526798] [gbkey=CDS]
MKESNLCPRFEKGMQLISKRWAGLIVYQLLSGPQRFCTIESNLPLSGKILSERLKELEQEGIIRRNVYPEMPVRIEYSLT
EKGLALEPIIRGVEQWSSAWLQPE
>lcl|NZ_CP026520.1_prot_WP_042227476.1_4027 [locus_tag=PC41400_RS20345] [protein=Rrf2 family transcriptional regulator] [protein_id=WP_042227476.1] [location=4526903..4527328] [gbkey=CDS]
MTISSRFAVAVHILSLLELNKEGRNTSEYIAGSVSTNPVVIRRITGMLNKAGLTRTSPGVAGAALGRTPAEITLLDIYRA
VQADMQDELFAMHENPNPDCPVGKNIQSTLETQFNRAQLAMERELDGVTLADIVADLARCE
>lcl|NZ_CP026520.1_prot_WP_042227283.1_4028 [locus_tag=PC41400_RS20350] [protein=AzlD domain-containing protein] [protein_id=WP_042227283.1] [location=complement(4527458..4527784)] [gbkey=CDS]
MEVSELVLWIIAGSAAVTWIPRVLPLLVLSRMKLPEWGIRWLQHVPVAVMAALLAQELLLTDGRLSVADNAYKLLAALPA
LAVAVYTRSLFGTVIAGVLSLAAFRLFF
>lcl|NZ_CP026520.1_prot_WP_042227285.1_4029 [locus_tag=PC41400_RS20355] [protein=AzlC family ABC transporter permease] [protein_id=WP_042227285.1] [location=complement(4527775..4528506)] [gbkey=CDS]
MNTNRPVPETENRSDGDSFLQGVNDCVPTLLGYLSIGFAAGVVEKTAGMSVAEIGLISLLLYAGSAQFIAAGMIAAGTPA
VSVIVTLFFVNLRHLLLSAALAPYFSKYTWWKNIATGTLLTDETFGVASSRAAQNGEIPYKWMLGLNVTAYLNWFAANMA
GGLMGERIAHPEKLGLEFALPGMFIGLLILQMAARRKIRTDLAVVVTATAVVVGGSRFLDSGLTVIAAILVAATAGMVIE
KWK
>lcl|NZ_CP026520.1_prot_WP_241688911.1_4030 [locus_tag=PC41400_RS20360] [protein=MarR family transcriptional regulator] [protein_id=WP_241688911.1] [location=complement(4528493..4528963)] [gbkey=CDS]
MRKMSTRIVLFQQNAAHSLGVIHTDLKTADILNEKGPITAGELAKLTGLTTGTVTALIDRLEQAGFVRREKDPHDGRRVI
IVPVKEKQKTIGALYKPLGTMTREVCEAYTPEQLDLINTFLSRMVGVFEQANSAVSGPKPDDGCCKNGRSERHEYE
>lcl|NZ_CP026520.1_prot_WP_042227289.1_4031 [locus_tag=PC41400_RS20365] [protein=MDR family MFS transporter] [protein_id=WP_042227289.1] [location=4529193..4530716] [gbkey=CDS]
MIQKDNNLNRVVAGLLLAILMSAMDNTIVATAMGTIVSDLGGFDKFVWVTSAYMVTTMAGMPIFGKLSDMYGRKRFFIFG
LIVFLLGSALCGIAQSIVQLSIYRAIQGIGGGALLPIAFTIVFDIFPPEKRGKMTGLLGAVFGSASVLGPLLGAYITEYI
SWHWVFYVNLPIGLVSLLLIMRAYKESPGHVEQRIDWWGAITLVVAVVSLMFALELGGKEYAWDSVPIVSLFASFVVFFI
LFFIIERKAKDPIISFWLFKRRLFATSQLLALLYGATFIILTVYIPIYVQAVFGGSATNAGLILTPMMLGSVAGSAMGGI
FQTKTSFRNLMIISVFSFLAGMYLLGGLTPETNRALLTVFMIIAGFGVGFSFSLLPTASTYDLEPRYRGTATSTNSFLRS
FGMTIGITIFGSIQANVFTDKIKTAFQGMNGGGMAGIDMSNPQQIFQADKRALIPPDILNKIIHALSESIAFTFMAALIP
IVLAVIIVFLMGNARVVPAKQKTGQPS
>lcl|NZ_CP026520.1_prot_WP_042227292.1_4032 [locus_tag=PC41400_RS20370] [protein=PAS domain S-box protein] [protein_id=WP_042227292.1] [location=4530869..4533493] [gbkey=CDS]
MTSLQINPQSLADHLFASTADGLALYSPNGVWMQVNEAWYRLCGYSPEELARLTLKDLVHPDDWKDARPEAPLPDSYVQK
VRFFHKSKREIPVLLTISRLTGNSPETSQCYVLQASELSPVQDDVQTWNRIKRQYRLMSEHSRDILYHTAPDGTILDCAP
TIRRILGYSPDELIGNTEDVLYCFPSGLQTEQSDSAVNEEYARQVKLLHKDGRHIWFDLTVKTIIGDNGEELILKIGQEI
SDRKKNEQFIAEAQRIALMGSWEWDISSNEILFSDQFFRVSGIDPASAAVHGILSLVSEDFHPVFKEELDKALVNKELSF
EYRRRKETGEDIYLHIRGLVTYSDQGIPLKINGTIQDITDRKLVERKLQETVERYTSLKKYNHDAVFSLDLEGKILNTNL
MAQEMTGYAVHEMIGVCFTKLFGTEDITEILEASLKDTSADKHIDKITDKYGNSVEVLTTIAPIIISEANVGYYIIAKDI
REQKKLMIEKESAEATNKAKGEFLAMMSHEIRTPMNAVIGMTDLLLETTSFNSEQREYLNIIRKSGDTLLGIINDILDFS
KIESGVTELVEEPFDLRTCILESVDLLTPKAIEKNLEMICAMSQNMPSTLLGDAKRLKQVLINLIGNAVKFTSTGNITVS
VKKQAQSQGTVDLLFQIKDTGIGIPKEKADKLFQPFYQLDNFMTRETEGTGLGLAICKKLVQLMGGDIWIEQSDEPGAHF
LFNVVLKEAAQEAVPFPKDKTAAEKASGHSLKILVAEDNRINQLVLVKMLTNQGHTVRVVDNGSEVVEAALTERYDIVFM
DVHMPEINGLDAARMIKEALRPEEYPIIVAVTANALKGDREKCLAAGMDDYISKPIFSRTVNETIQKFFSFQQA
>lcl|NZ_CP026520.1_prot_WP_042227294.1_4033 [locus_tag=PC41400_RS20375] [protein=PIG-L family deacetylase] [protein_id=WP_042227294.1] [location=complement(4533579..4534280)] [gbkey=CDS]
MAPVVAFIYAHPDDESFGSACLIRSLADRGIRSVLLCATRGDAGRPGPLGPMTPEQLADAREVELQTACDLLGVSDVMHL
GLPDGKLAGMDRPALADAVAEFLNETGASVVVTFPEDGISGHPDHIAIHHAVNEAVLGGRCPHVQKYYYNLPSYRDMADA
PSVVRLETAPYWKIKAQALLAHRSQRFSVEKAFGGSETLQESKDFPYESFGLVWLRGEKNPVRKEEYVLDDLI
>lcl|NZ_CP026520.1_prot_WP_042227297.1_4034 [locus_tag=PC41400_RS20380] [protein=Fe-S cluster assembly protein HesB] [protein_id=WP_042227297.1] [location=complement(4534321..4534596)] [gbkey=CDS]
MQIKVTEAASDLFRKDWGFGDGDHVRIFVRYSGGGEDAFSFGIIKDTPHYPAVTAEAGGITFFMEEKDVWYLDGKDLRVD
GIAEEIVLNRL
>lcl|NZ_CP026520.1_prot_WP_164977147.1_4035 [locus_tag=PC41400_RS29920] [protein=hypothetical protein] [protein_id=WP_164977147.1] [location=4534803..4534976] [gbkey=CDS]
MEQPKGQEVRNEKNPDHNSPEQERKVDRKQDIEPQAEEWNTRNKISGTHQDENSRNL
>lcl|NZ_CP026520.1_prot_WP_042227299.1_4036 [locus_tag=PC41400_RS20385] [protein=aminopeptidase] [protein_id=WP_042227299.1] [location=complement(4535059..4536288)] [gbkey=CDS]
MTSFQKQLEKYAELVVKVGVNIQPGQILYIESPIESVEFTRHVVRKAYEAGAKFVEVNWSDEQTTRIRYESAPDESFDFY
PEWTARMMEQLAEGNGALLNIKVPDPELFKGIDTGKISRAGKAASIARQRFQGYVRNHKFSWCLVKAPTKAWADKVFADL
PEDERVSVMWETLFKMNRVDKDDPVAAWKEHIAQLSETHERLTGKKYRRLHYKAPGTDLSVELPEGHLWLGGGGENEQGD
YFVANMPTEEVFTMPKRDGVNGTVSSTMPLNLGGRLVDKFKLTFKDGIVVDYSAEVGQEHLKSLFDTDEGAKRLGEVALV
PHDSPISNLNRIFYNTGIDENASCHFAVGSAYPFTLENGTKMSQEELLARGANVSLTHVDFMVGSADLDIDGELADGTIE
PIFRKGNWV
>lcl|NZ_CP026520.1_prot_WP_042227301.1_4037 [locus_tag=PC41400_RS20390] [protein=SulP family inorganic anion transporter] [protein_id=WP_042227301.1] [location=4536630..4538084] [gbkey=CDS]
MLRQLNRTWFSNPRSDILSGITVALALIPEAIAFSIIAGVDPMVGLYASFTMAFVISIFGGRPGMISAATGAMALVMAPL
VKQYGLEYLFAATVLTGTLQYILGLLKLGKLVRFVPQPVMTGFVNALAILIFMAQLTHFVGQGWVMYALVGVTLAIIYIL
PRFTKAMPSALAAIILVSIVTISFGIGVNTVGDMGRITSSLPLFHVPQVLFTWDTLKIIFPYSLSLAFVGILESLLTATI
LDEMTDTPSDKNREVRGQGAANIVTGFFGGMAGCAMIGQSVINIKSGGRTRLSTLVAGLFLLFLILVLGPVVEKIPMAAL
VGVMMMVSIGTFDWKSIPRLTKIPLSDAFVMIVTVAIVVATHDLSKGVFAGVLISAVVFAWKMAKIRATAKVSPDGTKTY
TVSGQMFFGTMTHFVEQFDYMNDPERITIDFSHSHIWDHSAVTAISKVMLKYDLLHKKVSITGLNEESRRLVERVGLAAS
GGHG
>lcl|NZ_CP026520.1_prot_WP_042227302.1_4038 [locus_tag=PC41400_RS20395] [protein=YpdA family putative bacillithiol disulfide reductase] [protein_id=WP_042227302.1] [location=4538341..4539330] [gbkey=CDS]
MEEVIIVGAGPCGISAAVELKKAGIPALVIEKHCIVNSIYLYPTHLQFFSTPDLLEIGDIPFTTPSDKPSRQEALFYYRN
VARHFDLNIRSYEEVTSIRKKDGHFVVTSVNRGGRQIETEAKHVVISTGYFDTPNLLGIPGEELPKVTHYYREAHPYTNM
KVAIIGGNNSAIDAALDLMRVGAEVTVVYRQEQMSPSIKPWVRPIFESMVAKGHIRMLYASRVVRIDDLTVTVDTNGSQT
VLDNDFVLALTGFRPDRTLMIESGVELQGEHLKPLYDPETMESNVPGLYVAGVIASGHNANEIFIETGRKHGVAIARHIA
SKGHTDLSS
>lcl|NZ_CP026520.1_prot_WP_042227303.1_4039 [locus_tag=PC41400_RS20400] [protein=DUF6483 family protein] [protein_id=WP_042227303.1] [location=4539395..4540117] [gbkey=CDS]
MFERKDYLLTMITQMAEAIATRISKLRKEKKHAEALAELGELYGRMQLPPSKLLGSVPEEDALRMMMTGASLSGEKLLAA
GKLLREEADIYEEMSGEREAYQRRVNALYFYLTADELGAKSDDVSLHAEIAYLNHRLKAYRQPAAVLYKLMEYCCAAGSY
DKAENLLYELKGRDSSGDGELPERGLAFYRRLLACSDEKLQAGKLPRDEVEAGLREWQTAWPRLSAGGTGIAPSGAESAG
>lcl|NZ_CP026520.1_prot_WP_042227304.1_4040 [locus_tag=PC41400_RS20405] [protein=discoidin domain-containing protein] [protein_id=WP_042227304.1] [location=complement(4540300..4542957)] [gbkey=CDS]
MKKIGVSLAVSFAACAALGLSASAAQPHSSYWYPEQLLKWSPGADPDAKFNRGSIPLQERFTGDKVNANASKDPKIVALS
ALNAGTSGVPSQGSDKFAANTFGYWQYVDKLVYWGGSAGEGIIVPPSADAIDAAHKNGVPIMGTVFFPPSVYGGKYDWVK
QTLQQKPDGSFPVADKLIEVAKTYGFDGWFINQETEGGTPADAQKMKEFLKYLEARKSPSMQIMWYDSMTKDGAISWQNA
LNDKNEMFLQDKQQKVSDSMFLNFWWRDLAGSAAKAKSLGRSPYELFAGIDVEAKGYDTGIQWDMLFPQGKPAATSLGIY
RPDWAFNSAENMKDFFARENKFWVGQNGDPSRTDTNQAWKGMAHYVAEQSPVNRLPFVTHFNTGSGQQYYVDGKLVRDKG
WNNRSLQDILPTWRWMAESKGTALKPSLDWSDAYYGGSSLKVEGTLSPVNATTLKLYRTDLAIEPATKLSVTYKTSQKPG
MKVALTFADPKDEVVYLDVKDKVQPGWTTETFNLTPYKGKRISSISLVFDSKETIPAYAINIGKLSVSSVKQEQSSSLPA
VKDLKVTGSDFRDGIYGDARLEWTALDQEVKQYEIYRVLPDGKEEFLGATPNNVYYVPEMKRSNKEDTTVLKVVAVNGQY
RAGQAAAAAIKWPAYPKPAARFKTDKTLVAPGEQVTLIDQSTEVTEEWLWTIDGASPSTSTDKQPVVTFAEEGVYSVTLT
AKNSAGQDSVTKQALITVRKDAGAAVNLALGKTAKADNACGPAEGAANAVDGKVTGNSKWCALGNLPHWLTVDLGAEKTI
SGFVLKHAEAGGEWAGFNTSDYTVQVSADGVTWTDVLEVKGNRAGETNDAVALVKARYVKLNIVKASQGGDTAARIYEFE
VRGLQ
>lcl|NZ_CP026520.1_prot_WP_042227306.1_4041 [locus_tag=PC41400_RS20410] [protein=ABC transporter substrate-binding protein] [protein_id=WP_042227306.1] [location=complement(4543279..4544772)] [gbkey=CDS]
MAGKKALVFMFVLMLALSSLLAGCGSKSGGGDSKQGAGDSTEKEVKLKVYLIGGPQRDLPKVQEEINKYVKEKINATLDI
TMIDWGDYSKRMQVLSASGENFDIAFTSSWAFEYRTNAAKGAFMPLNDLLDKYGKGIKEELDPRFLEGTKIDGKNYAVPV
NKELGEQWVWRFNKKYVDKYKIDITKVRTLQDLEPYLKQIKEGEPEDITPLAVPKGFKPYIPYDFILGDEFPVAMKIQDN
ASDFKFVNILETPELKASLETIRKYYQAGYLRKDVATLDGIDNIKTGKWFVDREITQPYADLGWSRSAGYDIVSTPMQDP
VVFTGSASGSMHAISANSKNPERAMMFLNLLNTDKYLRNLFSFGIEGTHYKKVSENVIEDLPAMKDGFQMPGFALGNMYL
TYLHKEEPADKWEAFKKFNDSAKVAPTFGFNFDVDPVKTEVASISAIAREFYPAIMTGSVDPNEYLPKAMEKMKAAGLDK
VIAEAQNQFDKWKAEKK
>lcl|NZ_CP026520.1_prot_WP_042227307.1_4042 [locus_tag=PC41400_RS20415] [protein=carbohydrate ABC transporter permease] [protein_id=WP_042227307.1] [location=complement(4544864..4545820)] [gbkey=CDS]
MSDLSGSVKAGAAAQARKGKKPKDLNAISPFWNFVFSTGVGLFALSCVFPFLFIIIISFTDEKTLAIEGFSLVPSQLSTA
SYRFLMNSGQQLFQSFGVTLLVTVLGTLLTLYLVATYAYALSRKSFEQRRFFSFLAFFTMLFSGGLVPSYIVVTQFLHLR
DSIWALILPSVLNAFYILVMRTFFSTTVPDAIVESAKIDGATEFGVFTRIVLPISLPGIATIGLFSTLAFWNDWFNALLY
IDKPTLIPLQTMLMRIQNNMDFVVQNSSKVVSYDIAASLPTETVRMAMVVLATLPIALAYPFFQKYFIQGLTIGSVKE
>lcl|NZ_CP026520.1_prot_WP_042227309.1_4043 [locus_tag=PC41400_RS20420] [protein=sugar ABC transporter permease] [protein_id=WP_042227309.1] [location=complement(4545860..4546789)] [gbkey=CDS]
MPGIVKSLVRNRAFLLLVLPGAIWFIIFAYLPMFGTIVAFKDFRIHPDGFFASVFNSEWVGLKNFEFLFSTNDAFIITRN
TILYNLALIFLGLVCAVGFAIVMNELLNKRLTKVYQTAMFMPYFLSWVIISYFVLIFLNMEKGVFNQLLVYFGADPVNWY
SEPKYWPYILIFMGIWKNIGYGSVVYLAAIAGIDRSYYEAAMIDGASKWKQIKYITLPQLKPLMIILTVLALGGIFRSDF
GLFFQVTKNSGALYPVTDVIDTFVYRGLTVMGDTGMSTATGLYQSVVGLILVLTANYAVRKIEKEYAVF
>lcl|NZ_CP026520.1_prot_WP_042227310.1_4044 [locus_tag=PC41400_RS20425] [protein=response regulator transcription factor] [protein_id=WP_042227310.1] [location=complement(4547031..4548587)] [gbkey=CDS]
MRKLFFADDEPLIVQGLYNLLPWEQFDLQIAGSARNGLEALEKLKEEPVDLLITDIMMPEMTGLELLRRVKLLNPRTKFI
ILSGYDEFKYVKEGILLGIENYLLKPINIDELEATIKHILRDWEREEISRFCLDEDWKILRNNILHRWTNGNIDNKEFRE
RARMLGIPIDHEVYRSAALRIVTENEPAFQFYRLPGLAEECERIGTEELPENCEVIAFPGDGDEITLLFALSGNEVKVLP
ALRRMSEWIAGFTGFRVWCAAGQPEELYQGYPASARAVQAWYQAHLMNEQPGVIVDDLAAYAFREEERPALQEPAMDRFV
KLLAEGRRDEIDEFADLFFVRGDSVHPVPRQPYFNAAIQLMLAVKEMEKQPDYSEVFAALFRIQTRSGLASQVKRVAGGM
LDSQSDAVREYSTHVAAMLEQVKEYYSRELSLKTLSQKLHMHPNYLGQLFQQEMGTSFSDYVNQYRVERATHLLLHTDRK
TAEIAQEVGYWDSSYFYRTFKKYAGVSPTELRSMYIKR
>lcl|NZ_CP026520.1_prot_WP_042227312.1_4045 [locus_tag=PC41400_RS20430] [protein=sensor histidine kinase] [protein_id=WP_042227312.1] [location=complement(4548584..4550419)] [gbkey=CDS]
MPSLKTFYRIHLKKRMFNKILLFYSIVMVLLFMGVSILAYQYYEQRSVREQTDRVLKDLDIVSVNLNQLYERMYSAIQQL
YTDAAVNEDLTYFLSHEYEDYLKRRLNQYIRSSGTEPKNFDSQLRSFLKDEPSIANILLYSKGERFFYFMNPRTQRFYYP
SEIPQDSPGWFDSTINHQWLPTDQRSVLWEARKDSGPVYSYTHVLKDTVSLQKFGAIVFDLNTDEIKRLVESKLKDRGAK
MLLMTSGGQVIYDSENRYQGQIYPYWQQLHEQQDWVELEEPSKVNVLTIGNTGMTIAGIIPKSRIEQSMDTIRLSLIGIT
ALCIAVSFMFTFTIIRQYSKKIRKIVTHIRRLQEGDLSARIVLTGEDELQDISQNFNYMCDRLETYIDQVYRSEIKQKNA
ELVAMQAQINPHFLYNTLESIRMKAITSGARDVGQMIYILAALFRDMIRKQTNVTLAEEMEMNVMYLKLFQYRFENRLEV
ETCLDKNAANCQVVKLLIQPVVENYLVHGFRPESDSNRIAVSMTREEENIVVRVADNGKGMPPERLAEIRKRLAMKPDSG
GLGGDSLGLVNVNERIRSHYGSDYGLAIDSGPDQGCEVMLKLPAIQGEMLL
>lcl|NZ_CP026520.1_prot_WP_042227313.1_4046 [locus_tag=PC41400_RS20435] [protein=alpha-mannosidase] [protein_id=WP_042227313.1] [location=4550610..4553762] [gbkey=CDS]
MFLTEHKLQSRIRELSELRYRDNHPIPSFLSLEDQRKEINPSLPAPAKDWKTMNVGNSWKGRDLYLWLSAEIHIPEAWSD
KRILGRFDFGDTGGGNNSGFESLFYWNGKPYQGVDSNHKEVFLPDDAAGTSRQLLFRLWSGLEGGGQPQDQTHAIRQAEF
CWLDEAADDLYFTAQAALDTVRILDANAPERTALLHALNQSFKLIDWTEPGSEDFYVSVNAARTDLNTRIEGMDKRSDVT
VTCVGHTHIDVAWLWRLKHTREKCARSFSTVMRLMEMFPDYTFLQTQPQLYEYVKEDYPELYEAIKKRAAEGRWEAGGGM
WLEADCNLTSGESLVRQLLIGTRFLADEFGTECKYLWLPDVFGYSWSLPQILKKSGIHTFMTTKISWNQYNRMPFDTFHW
RGIDGSEVLTHFITTPEPWSGPDSWFYTYNGQIIPKTVKGIWDGYRDKEMNQELLLSYGYGDGGGGVNRDMLEMRRRLNS
LPGLPNVETGRADAYFERLQETAANTDAYVHTWDGELYLEYHRGTYTSQAYNKRMNRKLELAYREAEWLNAMQSVLHNDW
SLYKGKELTEGWKIILRNQFHDIIPGSSIREVYEDSVIEYEEAEAIGRTVDTQAVEAIAAGKDSSGASAYTVWNSSPWTV
TELVTIASPQQGESGQWLDLQGSVLRAQLHDEVWTVEVKNIPPMGYTTVKFAAADSRQAASASPFAWNHSGIETPFYVLE
WNAQGQLTRIYDKEQRREVLAPGERGNVLQLFEDKPLAHEAWDIDLFYQENMQEIRDFKGAELVEGGPLRAVIRFNWVYR
NSSISQDLVVYGGHPRIDFVTEADWHEQKQLLKAAFPVDIRTTEATFDIQFGNVKRPNHWNTSWDWARFETVGHQWADLS
DRGYGISLLNDCKYGYDIKANVMRLTLIKCATHPDTHADQGHHAFTYSLLPHRGDWLQGQTVKQAWMLNSPLRASEGTAE
VPSRSMFTLSGDTAMISAVKKAEDSGKVIVRVHDYSGSRNALELTSDFRIASWRETNLMEKPEGPVVHDGTISFVLQPYE
IKTFEIDLEA
>lcl|NZ_CP026520.1_prot_WP_042227315.1_4047 [locus_tag=PC41400_RS20440] [protein=alpha/beta hydrolase family protein] [protein_id=WP_042227315.1] [location=complement(4553867..4554646)] [gbkey=CDS]
MAFIQCDFYSEVLGMSTSIQVILPQRTTGQIGLEGSASGLKHPTLYLLHGLSDDDTIWMRRTAIERYAAPLGLAVVMPNA
ARSFYTDMESGYDYWTFLSEELPQVCRSFFPLSDRREDNFVAGLSMGGYGAFKWALKKPDQFAAAASLSGVTQVKEIEKR
FPKDFRLVFGKNKELLRAGELFALARYAAEKKTELPALYQCCGTDDFLYEENIAFRDYVQSLNIPLLYEEEEGRIHDWSY
WDQQISRVLEWLPLKRQTE
>lcl|NZ_CP026520.1_prot_WP_042227477.1_4048 [locus_tag=PC41400_RS20445] [protein=methionine ABC transporter ATP-binding protein] [protein_id=WP_042227477.1] [location=4554977..4556005] [gbkey=CDS]
MIEIRKLRKQYGSGSRITHALQEINLTIDKGEIYGIIGHSGAGKSTLLRTINLLERPTEGSVFVDGVDLTALSTKKLQDQ
RRRIGMIFQHFNLLSSATVEENIAFPLRLAGTAKSEARRRVGELLALVGLTEHGKKYPAQLSGGQKQRVGIARALATNPH
VLLCDEATSALDPQTTNSILNLLLDINRKLKITIVLITHEMQVIRSICDRVAVIHDGQIVEQGPVAEVFLKPRHDVTRQF
IEQVADSVEIDKPDVSGAPAQDGTVLKITFLGETTYEPILSRTVQQTGVTFSILQGTISRMKNTPYGQLVVRLQGPRDAA
ENTIRLLREHGLDVEVYSNGLV
>lcl|NZ_CP026520.1_prot_WP_042227317.1_4049 [locus_tag=PC41400_RS20450] [protein=methionine ABC transporter permease] [protein_id=WP_042227317.1] [location=4555992..4556651] [gbkey=CDS]
MDWSKIDWSEIGIATNDTMMMLGLSALFTFVIGLPIGVLLYLTSKGSLLENRTFNIVISFVVNVLRSFPFIILMIAAIPL
TKLLVGTSIGVKGTIPPLVIAAAPFFARLVETALREVDRGIIEAAQSMGAGRFQIVRRVLLPEALPGLIAGITITTVTLV
SYTAMSGMVGGGGLGDLAIRYGYQRFQTEVMIATVVILLILVQILQMIGDRLVLYFTRK
>lcl|NZ_CP026520.1_prot_WP_042227319.1_4050 [locus_tag=PC41400_RS20455] [protein=MetQ/NlpA family ABC transporter substrate-binding protein] [protein_id=WP_042227319.1] [location=4556687..4557532] [gbkey=CDS]
MKKRSLIVLFLIVALTTLAACGKQKAEPSTTATPASGAGEKAAVTLKVGATAVPHAEILESVKDKLKAEGVNLEIKVFTD
YVLPNTQLSEKQLDANFFQHLPYLEEFNKSKGTKLVSVGTVHIEPFGGYSRKVKAIGELKDGATIAIPNDPSNGGRALAL
LEKNGLIKLKEGVGIAGTAKDIVENKKNFKIKELEAAMLPRVLDEVDLALINTNYALEAKLVPAKDALFVEDKDSPYANI
IAARDDNAKSDAVQKLIKALQTPEVKKFIEDKYQGSILPAF
>lcl|NZ_CP026520.1_prot_WP_042227321.1_4051 [locus_tag=PC41400_RS20460] [protein=sugar phosphate isomerase/epimerase family protein] [protein_id=WP_042227321.1] [location=complement(4557609..4558487)] [gbkey=CDS]
MSIRVPNPIGVIVDSFGLPVREGLQKAKDVGADGVQIYAVKGEMYPDNLTASQRKELKAYIDSLGLQVSALVGDFGGHGF
QVRAENKEKIEKSKRVLDLALELGTNIVTTHIGIVPSEKDDVYEAMQEACEELGQYAKSVNAYFAIETGPEPAAHLKSFL
DTLSTNGVSVNFDPANMIMVTGDDPVQGVYTLKDYIVHTHVKDGLRHKEFDPRILYGAVGFEPMAHERIAEMLREGELFE
ELPLGEGKVDFDRYFQALQEIGYEGYLTIEREVGDDPAEDIRKAVEFIKAYR
>lcl|NZ_CP026520.1_prot_WP_042227323.1_4052 [locus_tag=PC41400_RS20465] [protein=Gfo/Idh/MocA family oxidoreductase] [protein_id=WP_042227323.1] [location=complement(4558516..4559577)] [gbkey=CDS]
MSKLKIGVIGAGSISELHFGAYAKNAQVEIHAVCDLNEERAKEKAKQYGAVKTFTDYKQLLADPEIDAVSICTWNNTHAE
ISKAAVEAGKHVLVEKPLCKTVEEAHAVQEAVRKSGKVLQVGFVRRFGTNTNVAKRFIDSGELGSIYYAKATVLRRLGNP
GGWFSDVERSGGGPVIDIGVHVIDLCWYLMGKPKVKSVSAVTYDKLGNRSHIENLSFYKAADYDAAKNTVEDMASALIRF
ENGAALSVEASFTLHAKKDEISVKLFGDQGGAEIEPELMLVTEKHNTILNATPQIDHLTFDFEAGFRREIEHFVDCCLGK
HETISPVEDGLELMKILCGIYESSAAGKEIQVN
>lcl|NZ_CP026520.1_prot_WP_042227325.1_4053 [gene=gntK] [locus_tag=PC41400_RS20470] [protein=gluconokinase] [protein_id=WP_042227325.1] [location=complement(4559803..4561359)] [gbkey=CDS]
MNSMTRYMIGVDIGTTSTKAVLFEENGKVVAKAGGGYPLYTPTSSTAEQDPDEIFQAVLKSVRGVMDQARVLPGEVLFVS
FSSAMHSVLPVDPEGKPLMRCMTWADNRSAEWSGRLKKDLGGHELYLRTGTPVHPMSPLTKLMWLRHDHPEIFGKTAKFI
SIKEYVFAKLFGTYLVDHSVASSTGLFNLQALDWDAQALELAGITPDRLSRPVPTTHVVEGLSASYAAEMGLAPSTPFVV
GASDGVLSNLGVNAIEPGVVAATIGTSGAIRTVVDRPVTDPKGRIFCYALTESYWVIGGPVNNGGMLFRWVRDEFAASEV
EAAKRLGIDSYDLLTQIAGQVRPGSDGLLFHPYLSGERAPLWNPDARGSFFGLTLHHKKEHMIRAVLEGVIFNLYTVLLA
MEEKIGRPARIHATGGFARSALWRQMMADIFDQEVVIPESIESSCLGAAVLGLYATGRVDSLGVVSGMIGATHKHEPDKA
NARIYHKLLPIFIGLSRKLEEEYSAIAAFQQEVFKDRN
>lcl|NZ_CP026520.1_prot_WP_042227327.1_4054 [locus_tag=PC41400_RS20475] [protein=GntP family permease] [protein_id=WP_042227327.1] [location=complement(4561430..4562791)] [gbkey=CDS]
MNSIFGLSHSVTLLVWTLISIVFLVVFIAKFKWNPFVTLIISALMLGLLSGMKAQDVIKAVTGGLGGTLGTIAIVIALGT
MLGKMMAESGGAEKIATTLVDRFGEKRVHWAMMVVGFIVGIPVFFEVGLILLIPIVFTVARKTKMPLLQIGIPILAGLST
VHGLVPPHPAPMIAIDAYKADLGKTILYSLLIGLPTAIIAGPLFGKFIGKRIQAEPPAKLAEQFAVSENRNLPGFGITLF
TILLPVILMLIGSVANIVDPKGTSGLTVFCEFIGHEIIALLIAVVFSFFSLGFARGFKKEDISRFTSECLAPTATIILII
GGGGAFKQVLISSGVGAAIAEFATQANVNVILFAWLVAALIRVATGSATVAMTTAAGIVAPVLAQTPGANLELVVLATGA
GSIVLSHVNDAGFWMVKEFFNLSVPQTLKSWTVMETLLSVIGLAFIMILSVII
>lcl|NZ_CP026520.1_prot_WP_042227329.1_4055 [locus_tag=PC41400_RS20480] [protein=GntR family transcriptional regulator] [protein_id=WP_042227329.1] [location=complement(4562943..4563617)] [gbkey=CDS]
MRYPSAWLRGVSLGESIASELRLQIINGSIKPGEVISENRIASDFGTSRSPVREALKALSIEGLIRLERMGAVVLGLSQT
DVQELYDVRFIIESFVQDRLALANQASLIGKLNQIIDRMELAVKHGDPEDFAFQDITFHETIIEAARHKRVLHLWTSIRQ
IVITVLLITTDEIFSQGEEKLLDVIEKHRAIVRGLETHDPAVIGQVVRNYFADSRQTLETSLPS
>lcl|NZ_CP026520.1_prot_WP_042227330.1_4056 [gene=gnd] [locus_tag=PC41400_RS20485] [protein=decarboxylating 6-phosphogluconate dehydrogenase] [protein_id=WP_042227330.1] [location=complement(4563805..4564707)] [gbkey=CDS]
MRIGMVGLGKMGYNLVQNLLRHGHEAVVFDKDPAPGKRLAGLGAVPAENLEELVANLPAPRTVWMMVPAGKIVDSVIDSL
LPLLDAGDVLIDGGNSHYKESVARSEKLAAHGIHYMDVGTSGGTEGAERGGCFMIGGDKDAFEKIEPIFRDMAVDKGYFY
GGKSGSGHFLKMVHNGIEYGMMQAIAEGFEVLDKSGFDFDYEKVAEVWSNGSVIRGWLMELAGQAFSKDPKLEGIRGVMQ
SSGEGKWTVETALDLGANAPVIAMSLFMRYRSLETDTFHGKVVAALRNEFGGHAVVRSAD
>lcl|NZ_CP026520.1_prot_WP_042227331.1_4057 [locus_tag=PC41400_RS20490] [protein=hypothetical protein] [protein_id=WP_042227331.1] [location=4565151..4565450] [gbkey=CDS]
MSLWKNVRFIERDFWFQKMLNDTESLHSWQIDDLLGETNAQWDDLTFKFFDDGSVTIIDNDTDTRVSPQELKGAALDFYI
RKRIEFIRVSLQEKILMYA
>lcl|NZ_CP026520.1_prot_WP_042227332.1_4058 [locus_tag=PC41400_RS20495] [protein=DNA-3-methyladenine glycosylase I] [protein_id=WP_042227332.1] [location=complement(4565617..4566186)] [gbkey=CDS]
MELIRCGWVNQDPLYVNYHDHEWGVPERDDRKLFEMLCLEGAQAGLSWYTVLKKREHYQNVYDGFDAEKIAGYGEDKQAE
LLQDPGIIRNRLKVNAFVRNAQAFLKIQEKTGSFADYIWSFVDGQPVQNHWSSLQEVPASTPQSDAMSKALKKDGFTFVG
STICYAYMQACGLVNDHLTTCICHPSNQK
>lcl|NZ_CP026520.1_prot_WP_084706526.1_4059 [locus_tag=PC41400_RS20500] [protein=TatD family hydrolase] [protein_id=WP_084706526.1] [location=complement(4566219..4567151)] [gbkey=CDS]
MNKTMSESIKESIKEPIKEPMIDRKGFEPGAVGDRKAHDSEKLRYIDAHIHLEQYGEQERSLIFHDIAEGIPISMLLAVS
MNLPSCQRTRELYRRYPHIVRPAYGFHPEQPLPSEDELAELLAWMEQTVHEAAAVGEIGLPYYMRTEVEQAGQAFYLQPY
IDLLERFIAFAAKHNKPVVLHAVYEDADLVLDLLERYKIEHAHFHWFKGSEKTVERMIGLGYYISITPDISYEEEIRRLA
ARYPLELMMAETDGPWPFEGAYDGRMTHPLMVADVVRHIAGIKSVAVEEAAAQVYGNTCTFYRLQRQASG
>lcl|NZ_CP026520.1_prot_WP_042227333.1_4060 [locus_tag=PC41400_RS20505] [protein=ABC transporter ATP-binding protein] [protein_id=WP_042227333.1] [location=complement(4567234..4568001)] [gbkey=CDS]
MKKLELKEISKSFSTGGSDTKILNGINLHVNEGEFVSLIGPSGSGKTTLFHLIGGLIRPSSGQIVMEGSDVTGRKGLVSY
MPQQASLLPWRTVEENVILARETAGLPKAESLVLAREWLAKVGLQGYEKEHPHVLSGGMQQRVSFLRALLSPQELMCLDE
PFGALDALTREEMQHWLLKIWEQNKRSVLFITHSLEEALLLSDRIYLLSDKPSRIEREVIVPFERPRALTITEDPAFISM
KRELINLMKSVGSGS
>lcl|NZ_CP026520.1_prot_WP_009672896.1_4061 [locus_tag=PC41400_RS20510] [protein=ABC transporter permease] [protein_id=WP_009672896.1] [location=complement(4568014..4568775)] [gbkey=CDS]
MIRSLWKKGWPPLLTAILLLLLWQLGVTWAGTPAWRLPAPLQIVEEGIRQAPLLGKHVSFTVSITLLGFAIGSALGLLIA
CLLHVIPGMKAAFYPLLVLSQNVPTIALAPLLMIWFGFGLLPKLIVITLVCFFPVTVAALGGLSNPDPGMHRYLEMIGAR
RSQLFWKLELPYALPSVFSGLKISATYSVMGGVIAEWLGSDKGIGVFMLLAKSSFRTDRVFVAIFLIVVLSLLLFGAIVL
LEKWTVRWKPDKH
>lcl|NZ_CP026520.1_prot_WP_042227334.1_4062 [locus_tag=PC41400_RS20515] [protein=MTH1187 family thiamine-binding protein] [protein_id=WP_042227334.1] [location=complement(4568772..4569065)] [gbkey=CDS]
MANALVSIQIIPKTKNGEDVIPYVDRAIEVIQQSGVKHEVHPLETTMEGDLNELLDVVRKMNEAMIEFGSPNVISQVKIY
FNPEGASMDKLTEKYRP
>lcl|NZ_CP026520.1_prot_WP_042227480.1_4063 [locus_tag=PC41400_RS20520] [protein=ABC transporter substrate-binding protein] [protein_id=WP_042227480.1] [location=complement(4569155..4570192)] [gbkey=CDS]
MKRVIRTTALAAAFAVALAGCGKSGGNEGAQSSPEAAQTPAAVQKTDIKVVLDWSPNTNHTGLYVAKAKGYFAEKGLNVD
IIQPGAGGADAMVASGAADFGVSYQEGITQARTQSVPIVSIAAIIQHNTSGFASPAAKNIKKPKDFEGKTYGGWGSPAEN
AMIDSLMQTDKADPSKVKIVNMGDADFFTAVKRDIDFAWIYYGWTGIESELRKEPLNMVYLTDYSKQLDYYTPVLATSEK
VIKEKPELVKSFLEAASQGYQFAIDKPEEAAGVLSEAVPELDKNLVLASQKWLSPKYKDDAPRWGEQKKEVWENYAKWML
DHKLLEKSLEADKAFTNEFLPPAGK
>lcl|NZ_CP026520.1_prot_WP_042227338.1_4064 [locus_tag=PC41400_RS20525] [protein=patatin family protein] [protein_id=WP_042227338.1] [location=complement(4570449..4571297)] [gbkey=CDS]
MASLGLVLEGGGMRGVYTAGVLEYLMEKDWYVPYVIGVSAGACNAASYLSRQPGRNKKVTIGFVTDRRYISYRNLIREKS
IFGMDFIFDEIPNNLVPFDFQTFYGASHELVIGTTDALSGEAVFFRKNEAPEMSLSLIRASSSLPFVSQPVQYQERTLFD
GGITAPIPIHQSMKDGNERNIVVLTRPKGYRKTPFKQQWLARRFYPSYPGLVRAMVERSLVYNRTLDEIERLEGEGKVIV
IRPSTPFKVGRMEKDTSRLEALHDLGYKDAAEAKEKLREWLS
>lcl|NZ_CP026520.1_prot_WP_174446781.1_4065 [locus_tag=PC41400_RS20530] [protein=metal ABC transporter permease] [protein_id=WP_174446781.1] [location=complement(4571467..4572354)] [gbkey=CDS]
MISGWIILTGVLVACACSLVGCFLVLRNMAMIGDAISHSVLPGIVIAFLISGSRDSFVMMLGASAVGLLTVFLIQFFKQS
GVQADASIGVVFTALFAVGVVLVSLYAGHIDLDLDCVLYGEIAYVPWDTLTIAGIAVGPKAVWAVGLAFILGLLLIGMFF
KQFKLCSFDPAAAAAMGIPVALFHYLLMGLVSMTTVASFESVGAILVVGMLVVPPAAAYLLTERLSVMIPLSMGIGAASA
VLGYGLAYLLDASIAGCIICAAGGLFVLAFLFSPGQGVVARKWSRPRLSDNRKPA
>lcl|NZ_CP026520.1_prot_WP_042227340.1_4066 [locus_tag=PC41400_RS20535] [protein=iron chelate uptake ABC transporter family permease subunit] [protein_id=WP_042227340.1] [location=complement(4572347..4573297)] [gbkey=CDS]
MLTWLEALKDPNTQWILYGSSLLGLSGGVLGSFAYLRKQSLMGDALSHAALPGVCAAFMVTGSKSILGFLAGAAISGLLA
TLSIGWMTRHSKIKQDTALGIVLSVFFGFGIVLLTQIQHSPSGNQSGLDKFLFGQAASMVMKDVYTMAGVSVLLLLVCSL
FYKELKLTSFDTGFAKGIGYPTALLDQFVLLLILVAVVTGIQAVGVVLMAALLITPAVAARYWTEKLGTMVVLSGLFGAI
SGALGTLFSTIGSNMPTGPLSVLAATAWFLVSLVFGASKGLMPKLLLKWSAYRSVQKAKLPYESKRAQREGVSVGD
>lcl|NZ_CP026520.1_prot_WP_042227342.1_4067 [locus_tag=PC41400_RS20540] [protein=metal ABC transporter ATP-binding protein] [protein_id=WP_042227342.1] [location=complement(4573325..4574092)] [gbkey=CDS]
MQIQQLVQENRSKPPLLIEDMTVAYRKQPVLRQVTFAAPEGSLVGIVGPNGAGKSTLIKAALGLIPRVEGRVSVYGKPYR
EQRKRVGYVPQRESVDWDFPTNALDVVTMGTYGRLGWFRRPGGAERKLALDCLEKVGMADFASRQISQLSGGQQQRVFLA
RALAQKADLYFMDEPFVGVDAATERAIISLLEELKNQGKTVLVVHHDLSTVEEYFDTVMLLNVGLIAAGPVSEVFNRNNL
QKTYGGRLALLDRRL
>lcl|NZ_CP026520.1_prot_WP_042227343.1_4068 [locus_tag=PC41400_RS20545] [protein=zinc ABC transporter substrate-binding protein] [protein_id=WP_042227343.1] [location=complement(4574114..4575097)] [gbkey=CDS]
MNRLHERAAKRRGRWKGMSLVVLIAGSVIALSACSSEPSAMAGEKAVDAGKVRITTTIGMIADIAREIGGEHVEVTSLMK
SGVDPHLYKASQGDISKLDKADIIFYNGLHLEGKMTEILEQMGGKKPTVAVSDPIDRKELRSGDDAGTQYDPHIWFNVKN
WMTASEKVRDELIKFDSAHERDYRGKSDAYLAKLKELDSYTRSEIASIPEGARVLVTAHDAFGYFGDAYGIKVMGLQGMS
TASEYGSKDVTDLRDFLVANKIKAVFIESSVPQKAIDAVIEGAKQKGHEVKIGGTLYSDAMGEEGTNEGTYIGMVRHNVD
TIVKALK
>lcl|NZ_CP026520.1_prot_WP_042227345.1_4069 [locus_tag=PC41400_RS20550] [protein=hypothetical protein] [protein_id=WP_042227345.1] [location=4575777..4576094] [gbkey=CDS]
MVESQQHQQFIRHAVEAADAAILAAQEAEHELQGAITQADPNAIQRAQAKLGSAKRQVAEAAAQIHSFDGDRYGQQLQQT
IEQLQQAKQDLDVSEDHYHTPKQVR
>lcl|NZ_CP026520.1_prot_WP_042227346.1_4070 [gene=udk] [locus_tag=PC41400_RS20555] [protein=uridine kinase] [protein_id=WP_042227346.1] [location=4576217..4576846] [gbkey=CDS]
MLVIGIAGGTGSGKSTVARSIIDELGTKNVTLICQDNYYADRSELSMDERAKINYDHPNAFENELLLKHLGELRKGREIQ
VPIYDFTQHARSSETLHILGTPIIVIEGIHVLTDEHLRKALDIKVFVDTDPDVRILRRIMRDTNERGRSLESVVDQYITT
VKPMHDAFIEPSKKYADIIIPEGGENEIGIRLLTTLTESYMDTLLTDLN
>lcl|NZ_CP026520.1_prot_WP_042227348.1_4071 [locus_tag=PC41400_RS20560] [protein=MFS transporter] [protein_id=WP_042227348.1] [location=complement(4577079..4578275)] [gbkey=CDS]
MNVSSVKTMSPTRRLVLLLLVVIISGLSQGLLLPLLTILLEQTGVSSGVNGFNAAAMYIGSFASMFVVERILLALGYKKI
LILGIAMETVAILLIPVWGNLTFWFILRFIIGFGNSILHYTTQLWILTSAPPERRGRYISYYGMAYGVGFSIGPAGINLL
ALGSYAPFLTVALFYTVVILLLQRLPDQKVQSEGKESPAQKRYGTAAKIAWFALIPTFLYGYTEAVLNSSFPLYGLRLGL
TDTTVSMLLPLVGFGGLILQLPLGMLSDKIGRKPVLLAAAAMGAAAFLAVPFAGSSVWGIGFLLLMAGGMIGSFFSLGLA
YAADLLPRELLPSANVIGSILFSIGSVVGPNLGGLGIQYISLSSLFFLLGSVYVLFVLTGLAVRNRKRGEADAGLGQA
>lcl|NZ_CP026520.1_prot_WP_042227350.1_4072 [locus_tag=PC41400_RS20565] [protein=hypothetical protein] [protein_id=WP_042227350.1] [location=complement(4578364..4578912)] [gbkey=CDS]
MARGNTMVPFGYEEEPPAQRGTLLIFETFEDWSEDELGLIEAIADQRKFRKVVFYPQHEESLKRMGIPCASPYHKRVKLL
EELLRDMPVERYTVDTWEGKRKKYTPVDTSLRFLTEKFASPYFVMMNDRYAAEFAGYKGFDEWIRKVRLLVWKRFGVPLP
GKLLAYGDRLDFIGEKGESVEV
>lcl|NZ_CP026520.1_prot_WP_042227352.1_4073 [locus_tag=PC41400_RS20570] [protein=aminopeptidase] [protein_id=WP_042227352.1] [location=4579176..4580291] [gbkey=CDS]
MRDPRMVTFARNIIRYSVELQPGENILIELIGVKDPELAKCFIEEVYAVGGNPIVELRDPAINRSMMLGATRETLEKKAE
IELSRMKEMQAYVAVRAGDNVTETADVPEDRLKLYMSVYSRPVTNQRVNHTKWCVMRYPNGSMAQLANTSTEAFEDFYFQ
VCNLDYSKMETAMEPLVQLMNRTDKVRLVSPGTDLSFSIKDIPAIKCCGKRNIPDGEVYTAPVLNTVNGIISYNTPTVYS
GTSFENVVLKFENGKIVEATSNNSARMNEILDTDEGARYIGEFAIGVNPYILHPMKDTLFDEKIDGSIHFTPGQAYEIAD
NGNRSSVHWDIVLIQRPEYGGGEIWFDDVLIRKDGRFIIPELECLNPENLK
>lcl|NZ_CP026520.1_prot_WP_174446789.1_4074 [gene=infC] [locus_tag=PC41400_RS20575] [protein=translation initiation factor IF-3] [protein_id=WP_174446789.1] [location=complement(4580682..4581170)] [gbkey=CDS]
MLIMNEKIKAAEVELTGLNGENLGVMPREEALKLAKAANADLVCTSLMSSPPPCKLAARGTAKQAVQKDKQAARRQDMPL
KVKELRLTVHIEDHDYETKQRQAEKLLASGNAVQLVVRIQGKEGPKAKELLERLVKDLSASGTKETNIQVSGKQAAVRVN
PS
>lcl|NZ_CP026520.1_prot_WP_042227354.1_4075 [locus_tag=PC41400_RS20580] [protein=L,D-transpeptidase] [protein_id=WP_042227354.1] [location=complement(4581353..4582987)] [gbkey=CDS]
MGGATNRFSHPLEPHLVRVNKNLYISRGDPQYFDKVIRYADKTSPEAHFRMGERAREQGNLPRALFHYQEVMKTYPSPFY
SDANRAVHELAKREESDLAGVRKELEAATGSKFNKFLRTFLYALIVINLVLLVILLSGAGNKQTASAAENGKPSSYESHV
SQPQEGSSFSPEAASLRPEELSGVEPDERPYLLRLSPGTGIARAETILREEMELLIGRNPQKSRVIYAVRLTGQDALPGM
DGQVVPVKDPTEVEGAFAAAQYAPSLRKTAEITYLGEGAPRAQVLTRIGTQLIRTALGAYAADHGGKFPASLDELVRDYP
GNYLSFVPREAVTGSTRVSPSYDGGGGWVYRPAAADAGSALTPNLPGASLAGIILPASASAQANVPFEPVRVTVSKRDHR
LLLTAGGAVLEARAGIGRPESPTPDGTFAVRERVIAPAGRTPGVYGAAALGLGAIAVHGTADAASIGDGRSLGCIRVGDA
DMQALFALVPKGAAIRIAGAIPAGGASPGSGGLDLAAAAAALRPPGAPAVREQAPAGTVFHWLG
>lcl|NZ_CP026520.1_prot_WP_255246824.1_4076 [locus_tag=PC41400_RS30625] [protein=hypothetical protein] [protein_id=WP_255246824.1] [location=4583187..4583318] [gbkey=CDS]
MAGRNSKPKNRGPASQPGRLDQFGDKLADGKGPVPKPENVLGE
>lcl|NZ_CP026520.1_prot_WP_009672891.1_4077 [locus_tag=PC41400_RS20585] [protein=PH domain-containing protein] [protein_id=WP_009672891.1] [location=complement(4583404..4583781)] [gbkey=CDS]
MGILDGLLGNASEMSPEKLQKELEPFLIPGEQVQVAYKIIRDGFVFTNKRLILIDKQGVTGSKVEYMSVPYKSIIRFAKE
SAGTFDLDAELKIWLSSTHEPIVKKFNKSANINDVYAMLSRYVLG
>lcl|NZ_CP026520.1_prot_WP_042227355.1_4078 [locus_tag=PC41400_RS20590] [protein=DUF3900 domain-containing protein] [protein_id=WP_042227355.1] [location=complement(4583848..4584924)] [gbkey=CDS]
MKFEVKFISFFVIREEGTGKAYRHFQTLDGDAYEGSALEQFLGGEFARLAKRKVERNPQSEQVPTKIGRFITEPGCGLET
NPNYNMFARLRSASSRDDFLKESDELARAYIQTSSVRSGALITASAKLPQYFDEPFVFVMKCDFEPKIARITDESSLLHE
VEMAINAKNMKAIQYPYMPEVGMLEDWELKIHQPSHAKYFEDLLKFVEYEKSMPELLNHQVVNLVQQHIEQTLEAHPEQK
EAEERDLELWAASEKRELQEKWAPELVMEAANFLVEQQPDLGMTFKLDHLTVKGKLADYGRKVHFAKFGDQYVVLLSGSS
LHFEKGVSPVELLQPEDIQSVLARLEREAAQDNGDGDF
>lcl|NZ_CP026520.1_prot_WP_241688710.1_4079 [locus_tag=PC41400_RS20595] [protein=hypothetical protein] [protein_id=WP_241688710.1] [location=4585120..4585371] [gbkey=CDS]
MIYGTMLLETDIELFAAALSQTEVYIWGMDHNDVYVQIGRGFIEKYNQDQVKIRSSLNSCTMIYSREKCEFSAKTESAEE
QPK
>lcl|NZ_CP026520.1_prot_WP_042227358.1_4080 [locus_tag=PC41400_RS20600] [protein=hypothetical protein] [protein_id=WP_042227358.1] [location=4585626..4587029] [gbkey=CDS]
MNVELMGLHVVYVCFIALIIGFMLFRKDTTPVCLIGIFLLGLAATGSLPQSVGGIFTSLIYAMKELLPTILVISVIVGMS
QVLSLSGVSDTMISPAAKLIRSPGAAFWIIGIVMMTVSWFFWPSPAVALIGAVLLPAAVRAGLPAMGAAVAMNLFGHGIA
LSGDYVIQAAPKLTADAAGLPVGSVMEASIPLVAVMGAVTTATAFWFLRRDLRSGALKPESAIKPPAPVIGGTSSGGKLL
SGRRRLLLAVAIPLLFAADIAAMFKLNLQGGEATALIGGTAALILLLVSLMAHRGGGFGKTTAYLVDGFHFGFKVFGPVI
PIAAFFYLGDAALTELYGHVLPASSHGIVNDLGLALAQTVPVHPAASAVTLTAVGAITGLDGSGFSGISLAGSVASLFSS
ALGAGAATLTALGQVAAIWVGGGTLIPWALIPAAAICGVDPFELARRNLKPVMIGLAVTTVFAMFLI
>lcl|NZ_CP026520.1_prot_WP_042227359.1_4081 [locus_tag=PC41400_RS20605] [protein=metalloregulator ArsR/SmtB family transcription factor] [protein_id=WP_042227359.1] [location=complement(4587319..4587699)] [gbkey=CDS]
MSGTTEMTTMEYFEQCTNLFRTLGDPARQQIILLLGKHGPMNVKSITEHSHLSRPAISHHLKLLKQAEIVTIRQQGTEKY
YALDAGAAMKLMKDLIAAFEKECPPDAAGREEGEGLPCEGGASTGR
>lcl|NZ_CP026520.1_prot_WP_042227360.1_4082 [locus_tag=PC41400_RS20610] [protein=NAD(P)H-dependent oxidoreductase] [protein_id=WP_042227360.1] [location=complement(4587753..4588349)] [gbkey=CDS]
MTTQAPSRVLIIQGHPDRESYCTALAESYRKGAAAAGAEIRFVQISDLEFNPNLAYGYRKRTELESDLLGIQESIRWADH
VVFVYPTWWWTMPALLKGLIDRVFLPGFAFKYKSKDSFAWEKLLKGKSARLIVTMDTPVWYYQTISGSPGHKAMKKGTLE
FSGFKPVRVTTLGSLKLSTAAKRESWLDKAENLGRKLI
>lcl|NZ_CP026520.1_prot_WP_042227361.1_4083 [locus_tag=PC41400_RS20615] [protein=uracil-DNA glycosylase] [protein_id=WP_042227361.1] [location=complement(4588793..4589464)] [gbkey=CDS]
MTILKNDWAELLSEEFEKPYYLKLREFLKQEYAERTIYPEMHNIFNALHYTAFKDVKAVILGQDPYHGPGQAHGLSFSVK
PGVPAPPSLKNMFKEMKDDIGCPLPGTGYLKHWADQGVLLLNTVLTVRAGEANSHKGKGWELFTDKVIETLNRKETPVLF
ILWGSHAQSKTQLIDLNRHGVIKAPHPSPLSAHRGFFGSKPYSKANAWLREQGIEPIDWCLPE
>lcl|NZ_CP026520.1_prot_WP_129112518.1_4084 [locus_tag=PC41400_RS20620] [protein=type IA DNA topoisomerase] [protein_id=WP_129112518.1] [location=complement(4589481..4591973)] [gbkey=CDS]
MKTLVIAEKPDMGRTIAAVIEPKAANKRTYIEGSDYIITWAIGHLIELAEPDKYDEKYKKWRFEHLPIIPDEFKLLPNPR
TKDQLKVIKEVAKQCDKIVNACDAGREGQHIFSLIQRHLKLKQPVKRLWISDLTAETIRSGFQQLKDGSDFDNLTRAARA
RSEADWIIGMNGSRAFTTKHNVLLSVGRVQTPVLALIYDRQKLIEDFSSLKFYEVEAFFTQEQTEYRGLWQGERITDPAK
AEGIAAKVKGKPARIASYEVKDTKEYPFKLYDLTLLQRESNAKLGFSAKKTLDVAQALYEKHKVISYPRTNSNYVTEQNI
PEMHKTLSALQGTAYDEWVRQANRNLVHKNNKAICNPAKVEDHHAILPTGKRANGLSADEQKVYDLIVRRFLSQFYPAAE
YKVHTVLTEVEKETFKTTVKELLQLGWKVLYPESKKDGKRAKAKDDKGKEEDEAEEEETSEPFSLNASLGAHCRDAAAKE
KETQPPKAFTEGTLLKAMESAGKQIEDEELRDAMKDSGLGTPATRAATIERLKHVGYVTMQGKRITLTPKGRTAVELIRA
AGVELLTSPEMTGQWERRLNEISRGQASDETFMRNVKKFATLIVDKVRVQNRAARTAFEGDEPAGKAGGRAATGKTRRSA
KASGTDGAAAGTAPGARAAAGTAARESGAPAGLAVSARSGAAGGRAGQAPVRETAGAEGQASGRSSRTTAGQAAGQRSRG
ASAGQAAGPGASAAAKPADGPRVVGACPRPGCGGTIFMGKKGYGCSHYKTGCTFVIWKESFGRSLTDAQVKALLEKGKTA
KLKLERQGGEEVSARLVLADPSTGRLETEE
>lcl|NZ_CP026520.1_prot_WP_042233254.1_4085 [locus_tag=PC41400_RS20630] [protein=redoxin domain-containing protein] [protein_id=WP_042233254.1] [location=complement(4592557..4593084)] [gbkey=CDS]
MTHKLRRGAVLLIAGLIAAALAYTLLHADSKEPAVAVRTGAMAPDFTVTGLDGRSAALSDYRGKGVLLNFWASWCGPCTH
EMPRMNEAYLEGIPDVEIVAVNVGESRGTANEFAMRENLAFPVLLDPSGEAAQKYRVVGLPATFLIDPDGKIAAIRPGEL
TSSSEVKKLLQSLSR
>lcl|NZ_CP026520.1_prot_WP_042233252.1_4086 [locus_tag=PC41400_RS20635] [protein=ABC transporter ATP-binding protein] [protein_id=WP_042233252.1] [location=complement(4593100..4593840)] [gbkey=CDS]
MAENVLQVSDLHKKIGKQPIIHSVNLRISRGQVLTLCGGNGAGKSTILRMIAGITQPTSGVIEVNGRKWKENRTAYAQQI
GYMPDDYLFGQGLSAEETLVFWASLRKVSKARVLETLELVGLADVRRKPVSSFSKGMRQRLLFAQALLAKPALLIMDEPT
NGLDPYWMDSFVELVRYAKREGHAVLFSTHQLGVAESAADHVVFLKEGRIAEEGTAESFREAYGPGGLQAAFSKLFGYRT
GGDGGI
>lcl|NZ_CP026520.1_prot_WP_042233250.1_4087 [locus_tag=PC41400_RS20640] [protein=ABC transporter permease] [protein_id=WP_042233250.1] [location=complement(4593833..4594651)] [gbkey=CDS]
MNVLTIALREVKVGFRNPWAYSFMALFSLFSLSLLLINAQSFVEGYSSSTSSMLNLILYLLPLMTLLLGSFSLTAEKEEG
SWQLLSTYPLATFAFIAGKYTGLTVVMLTIVAFGYGLTGIVGALTGRGLQPATLGLFFAFSTGLVLLFLAIALVLGTFAK
NRWQALTAGVAVWFFAVIGWPTLLIAVLGQLPYVWIKSALVALTFLNPAELVRLFVVVKLGGGAALGPEYYEWVRWIGRP
EGTLVFMGLCLVWIAVCAGLANWIWERGRSRG
>lcl|NZ_CP026520.1_prot_WP_042233248.1_4088 [locus_tag=PC41400_RS20645] [protein=nitrous oxide reductase accessory protein NosL] [protein_id=WP_042233248.1] [location=complement(4594648..4595217)] [gbkey=CDS]
MKKTGLFLLSVLLIMTILSACGKAKDQAEPINEETDKCAICNMQIKDDAFAVQLKTKDGKTYKFDDLGCMNEWKGKNAAD
SIQVQFVRDYNNKEWIKYEDATYVYDPSFKSPMAYGIYSFKDKAEAQKFADSQKKGKVMTAADLGKHTWEQSKSMMNMGG
SHDHSVTPEAGGDKDAHGGKAGDQGTHGK
>lcl|NZ_CP026520.1_prot_WP_042233246.1_4089 [locus_tag=PC41400_RS20650] [protein=NosD domain-containing protein] [protein_id=WP_042233246.1] [location=complement(4595214..4596581)] [gbkey=CDS]
MKLRLIPLWLCVCAAAAAVLWLPHQRAEAAPPDAGAARPAELQQLIDRTPSGGLLRLPAGSYEGPAVITRPIVIEGQSGA
VVRNKDGSGKPAITVRADDVSLKGLQIEQDGEGASSDTSAVLVQGSGALLHSLDIRTDAFGILLRDTGHNRIENNSVVRT
GSGAAKLSERRNGIDLFNAHDNVISGNRVSSMNDGIYLESSHRNKVEANAVEHSRYGIHCMYTKGTVVRGNTGMYNITGA
MIMGVKEAEVSGNSFTKQSESVNSQGLLFFDVHNSKVTGNKVEGNRVGLYVEQSGSNEFTNNDVLQNFVGVQFLESENNR
FTGNRFIGNVIEAEANESESNAFSGNFWEAFRGLDSDGDGSSELSYALNPFFQRLTGAVPAYQLFFQSPGMQFLEGMYAS
DREGWTKDEAPLMKPGTGPEQKPLSGSAGNTTLVFGLLLLLASVSTIYYVGVRRG
>lcl|NZ_CP026520.1_prot_WP_042233244.1_4090 [locus_tag=PC41400_RS20655] [protein=FixH family protein] [protein_id=WP_042233244.1] [location=complement(4596578..4596961)] [gbkey=CDS]
MLKPISRFLLAVLLVFAVTACSSELKGQDPAKVKVDLVTEPSPVKTGEKAVVTAKITGLTTQEGARVQLDVRLADNSGLP
ELLDAKSSGDGNYSIEKTFDQPGKYAIYVHLYQGVLHITKKKVLDVS
>lcl|NZ_CP026520.1_prot_WP_009672845.1_4091 [locus_tag=PC41400_RS20660] [protein=hypothetical protein] [protein_id=WP_009672845.1] [location=4597155..4597502] [gbkey=CDS]
MFMVYGIALLLAGLLTINSIFSYQSKHIDPSLWQTLWFQCKMIPLVFAANVMIGYGVKFCYKTTGNLTLSLGISKGIEIF
ICVLMGYIFLKEIPNWKTWVGLGIVVGGFVITKLK
>lcl|NZ_CP026520.1_prot_WP_042233241.1_4092 [locus_tag=PC41400_RS20665] [protein=glycoside hydrolase family 31 protein] [protein_id=WP_042233241.1] [location=complement(4597783..4600218)] [gbkey=CDS]
MQTSETIHPDKSIDTVLEVAPKAPGRAARVTELDDSYVIHGQHANLLLTFLECGLFRIKLFHGNKPEFKAATVAVLSEAF
QPVKADRTESPEAFEFRTSALRLIVEKETFALTVQTPEGELLMKQPTLSWTTRGSLSGSYVMGTDSHFYGLGEKTSFLDK
RGERYAMWNSDVFAPHVPEIEALYESIPLLIHMHQTGTYGLFLDNPGRTDFDMRTHPDLFTIQCTTGEYDLYFIFGPQPK
DVVGRYTKLTGRISMPPKWSLGYHQSRYSYMNQQEVLELARTFRERNIPCDVIHLDIHYMDQYRVFTFDKDRFPDPEGMM
AELKKLGMRIVPIVDPGVKKDPKYPIYREGIENGYFCKKLEGELFIGDVWPGKSAFPDFTEDAVGKWWGEKHKFYVDLGI
TGIWNDMNEPAVFNESKTMDLDVIHGNNGDSKTHEELHNLYGMMMSKATFESLREQLGGERPFVLTRAGYSGVQRYAAVW
TGDNRSFWEHMAMAIPMVLNLGLSGIPFTGPDIGGFAHHSTGQLLARWTQMGALFPFCRNHSVIDSVRQEPWSFGEETEA
ICRTYIELRYRWMPVLYTLFHEASRTGMPILRPLVLEYPHDPQVTNLSDQFLFGSSVLVAPVYRPDTEHRAVYIPEGEWI
DYWSGEKHAGPKHILAHAPLHIMPMYIKAGAILAEEPLKQHADERTAASLTWNIYASGGSSEGTVYEDDGITYGYEQGAY
NVIRMSTEAADGGMKLAWAYEHTGLPDADRSFLIRIKLAGFVPAAVEGLQRLEAGVIDREQSAAGWTYDEAAGEILIRTA
DTPQQILHIRA
>lcl|NZ_CP026520.1_prot_WP_042233238.1_4093 [locus_tag=PC41400_RS20670] [protein=LacI family DNA-binding transcriptional regulator] [protein_id=WP_042233238.1] [location=complement(4600222..4601265)] [gbkey=CDS]
MNKITIKDIAAEAGVSIATVSNVMNGTGRVSGETAKRVLRVIDRHDYAPSTAARYLKERRSQLIGILVPTGDRGRLEDNP
FYWQLVMAVEAEAREERLHVIVSGVTEEDPLTFVQERQLDGIIAIGITGDSPLLGRIAQLDVPCVYLDCYLAGGENLQVV
NMNDRMGGYLGTKHLLSLGHERIAVIAGDIVTSGVYEERWLGYTRALEESGIAADRALLVETPVSLLGGYHSAQQVYMLS
PKVTAVFAFSDVAAIGLMKGLQELGVRIPDDLSVMGYDDIDYAGFMTPSLTTIRQDIGTKGKTAIRRILSMIRGSVPAEH
SVTLPVELKRRQSTARRFQSNSIQKEE
>lcl|NZ_CP026520.1_prot_WP_042233236.1_4094 [locus_tag=PC41400_RS20675] [protein=DUF5946 family protein] [protein_id=WP_042233236.1] [location=4601615..4602061] [gbkey=CDS]
MVDNRTVMESGRCMECGAEGIDGFSCYELFGFPLVWEHNDPKLYDLHFWLVSCYMIQHPSNYTEEGYRQLVRLFIDAYDH
DWDTAYILKRNRELMKNVDKLTNPLPNKDRQRKLAFWSMTIEDIYLGGEQQAIANIIRWKKQIRHELG
>lcl|NZ_CP026520.1_prot_WP_156972859.1_4095 [locus_tag=PC41400_RS29925] [protein=hypothetical protein] [protein_id=WP_156972859.1] [location=4602418..4602588] [gbkey=CDS]
MEFIKPVHVERSNVEWQVTDRIKAIVKYYSEYTGISENEVLDRFLVNILEDPDFKQ
>lcl|NZ_CP026520.1_prot_WP_241688711.1_4096 [locus_tag=PC41400_RS30425] [protein=hypothetical protein] [protein_id=WP_241688711.1] [location=4602662..4603909] [gbkey=CDS]
MSKRNLKRLSTFDDKVVQIETPIHEDEIHNRQNDYSLLPPQQFAKKYRSLLFQPVKITYKGKPFEINTNHCLDPYCKWFG
LPQRKFDDVKGKPHRYFIGSPKDGKQRINCNDDPINHGKGQLTLNCSTPTASNWSLAEEILRLETLNRTQSFNNKDYEFH
KENCSISIASPFHHPREFYRRGKSSSNSEKWQCKECKKITNVLPMRNQSFVYHQKRNDILHDLAKALINRTPVKRTCEML
KIGSSTYYNKLEWLYRRCLEFLERYETEAFSTKTFETMWLNTDKLVYNLNNVRKKGHGGKRYDNVEERILQTHIVVTSDV
QTRYVLRSDVAYDWNITLEQLEQDTVYYKDDHINNFSRKNERYRFPFAPQPPTLADTQSTLEYNDERAKFEKRKMYIDGL
HVNSSYTTYAQLWLI
>lcl|NZ_CP026520.1_prot_WP_241688712.1_4097 [locus_tag=PC41400_RS30430] [protein=hypothetical protein] [protein_id=WP_241688712.1] [location=4603937..4604626] [gbkey=CDS]
MITDDDASLYNSFFRVFSKEVSLGEAHHFITIVDKSKSLQEAFKEYQEGRKELTAWGISNGVVDSSISRVAYLKLKKQLE
NHTFHTEILRNGKTYRKWGNNKITHPLPSIDQGWYNVDCTSDLSSYEPKQIAKMILQVIDKSTNAFLQQIRRRLNILERP
LVTARGDKKSYIYSNFNPQYAQYAVTILRTFYIFCLPYRSSNDEQKLTPAQRMGLTNKIFSIQDILYFK
>lcl|NZ_CP026520.1_prot_WP_053228772.1_4098 [locus_tag=PC41400_RS20685] [protein=hypothetical protein] [protein_id=WP_053228772.1] [location=4604686..4605141] [gbkey=CDS]
MNNLECKMAIRHGFVDRYVDGEIRKCENNTSFFDFSIDGNSLFNTLQPYDFISVLSHKDEELQRNEFDKLLLNAPSGLES
GRYMIFVCPMCADLGCGAITVSIQREEEHIVWKDFSYEDGEKTKSLHLGTFIFEWANYTKTISSASFEKNE
>lcl|NZ_CP026520.1_prot_WP_129112519.1_4099 [locus_tag=PC41400_RS20690] [protein=hypothetical protein] [protein_id=WP_129112519.1] [location=4605210..4605539] [gbkey=CDS]
MYGIGDEEEKDVVPTFCKCGPQKPGYHCLANNCQFAGHAPAPYEIAYSHANGEVSHDAWIGFGGDMEPEPYNENEVTTLK
DTWQKVCKHKVDEAYNEYMIKTKLQDKSN
>lcl|NZ_CP026520.1_prot_WP_042233232.1_4100 [locus_tag=PC41400_RS20695] [protein=metallophosphoesterase family protein] [protein_id=WP_042233232.1] [location=complement(4605737..4606240)] [gbkey=CDS]
MKIGVLSDTHMRPTAGKLPDAVLKGLEGVDLILHAGDWTSPEIIEWVERIAPVESVAGNNDGPDIVKRFGYSKLIEAGGL
TIGLVHGDRGRGRSTEQRALFTFQADNPDVIVFGHSHIPYSERIDGLLLFNPGSPWDKRWQPRYSFGLIETDGEKTEARH
IFFDSRE
>lcl|NZ_CP026520.1_prot_WP_042233230.1_4101 [locus_tag=PC41400_RS20700] [protein=MOSC domain-containing protein] [protein_id=WP_042233230.1] [location=complement(4606275..4606835)] [gbkey=CDS]
MKKGNEQLAVMTGLLIADKPGTFITRHLDWAEMQFGGLVGDRHFGVTSKADVRQPMYPRGTEIMNRRQISIVSEEECEEI
AAALGIERVAPEWLGANMALKGLERLTALPRGTRILFPSGAGLVCEGENEPCVHPGKVIAEQTGNPKLAGKFVKAAWNRR
GIVASVERPGIVSAGDEVRILFPDEL
>lcl|NZ_CP026520.1_prot_WP_042233228.1_4102 [locus_tag=PC41400_RS20705] [protein=UvrD-helicase domain-containing protein] [protein_id=WP_042233228.1] [location=4607098..4609281] [gbkey=CDS]
MDKNQSAYQEEQNRLDAILADVNRRLAELQAVPVYRGPDFTEQALEAVRESSRQSLALASDEPYFGRLDFQPENAAGKSA
LYIGKFGVDDENGGEPHVIDWRAPVASMFYSYTGGDEKAAYESPDGLVEGLVYLKRNLVIRKQILQRVVDTYERDSDLLS
GAGGDEFLLYRLGENKDNKLRDIVSTIQAEQDKIIRSAKNAALIIQGVAGSGKTTVALHRLAFLLYQYRDQVKAENMIIF
APNSMFLDYISGVLPELGVGHIRQSTFADWATELTGLSFALENPADESAVWFALGSKRPPLNDEVSGRWKGSVAFRDFLE
SCFDGYARTCLPQEPFHPWDGKTLPAETIGVWYNEEYKHLPPAKRRERVVARIKRWITMELDKVWDAKQKKERKTKSTQR
LKAYLKGWPELDALSFYKAVFDPKSGAVWMTEELRGRIPEAVAKDTLASFRRKKVRHEDLPALVFIHASLHGIEGSERFD
HVVIDEAQDASPFQIAVLDQYTRQHSFTILGDLSQGIHAYRGTQDWAELTSLFGEDETAYFELERSYRSTMEIIRFANHV
LSIGVGAGLLAVPVFRSGERVKVLRVEKERESLFLRDAVERLLRRNYDTVAVITRTEEDAVRLHAQLGELGLAASRITAD
QKQYAGGISVVPVYLTKGLEFDAAILTDADPLRYGADAQDAKLLYVGCTRALHDLWVLYDGEPSPLVNVDDPEIVTRELP
GEDNAGS
>lcl|NZ_CP026520.1_prot_WP_042233226.1_4103 [locus_tag=PC41400_RS20710] [protein=DegV family protein] [protein_id=WP_042233226.1] [location=complement(4609551..4610399)] [gbkey=CDS]
MTIKIITDGTSDLPQELARELDITVVPLIVRFGDKEMTSDMSRTLFYELMRQEPALPQTASPAPHAFMEEYRKASQSDIL
VVSVSSNLSSTYHHALMAREQLMEEGYSGRIEVLDSRTASLGQGLLAVKAAKHAKEGMPFEELTARMKEEVKSHHTYFFL
DTLENVIKGGRLDRVRGAVASVLNIKQIMKASEEGTLEPFEKVRGTQNAIKRLIERVSEKVSETRHDLERCILGIAHSNC
EDRARDVRDQLLRLFPFGQVVITEMGPVIGTYAGEGGILISC
>lcl|NZ_CP026520.1_prot_WP_042233224.1_4104 [locus_tag=PC41400_RS20715] [protein=DegV family protein] [protein_id=WP_042233224.1] [location=complement(4610415..4611263)] [gbkey=CDS]
MAPIKIFTDSTNDLNPHYLKENEIGVVPLYVVFGDDSYRDGVDITPEVLYRKVEQSGKLPKTAAPSPADFEKAFRPHIEQ
GCDIVYIGLSSEISSTYQNAVLTSDLFPEGRVTVIDSRNLATGIGILICKAVEAAKQGRSAAEIETLVKSFIPKVSTEFI
IDTLDYLHKGGRCTSVQAFFGSLLKIRPVVQVKDGSMILTSKVRGKREKALEQLVDNALAFKHEMDPDVLFVTHSLAQEE
AELVRARLLQETGVKRVEITETGCVISSHCGPQTVGIVFIKS
>lcl|NZ_CP026520.1_prot_WP_042233222.1_4105 [locus_tag=PC41400_RS20720] [protein=amino acid ABC transporter ATP-binding protein] [protein_id=WP_042233222.1] [location=complement(4611311..4612033)] [gbkey=CDS]
MLKITNLHKHFGKLTVLNGIDIEIEKGEVVVVIGPSGSGKSTFLRCLNLLEVPTDGEILFEGISITNKKTDINSVRQKMG
MVFQQFNLFPHLSVKDNITLAPRKLKKMSVSQANETAADLLRKVGLAEKADAYPAQLSGGQKQRIAIARALAMSPDVMLF
DEPTSALDPEMVGEVLDVMKKLAKDGMTMVVVTHEMGFAREVGDRVVFMDGGVIVEQGRPEEVFSNPQHARTKEFLGKVL
>lcl|NZ_CP026520.1_prot_WP_009672909.1_4106 [locus_tag=PC41400_RS20725] [protein=amino acid ABC transporter permease] [protein_id=WP_009672909.1] [location=complement(4612026..4612715)] [gbkey=CDS]
MNFSFLNEYGHYFVDGTKVTLLLSVITVILGAIFGTGMSLLRLSGIWPLRWLATAYIEFFRGTPIIIQIFLVYYALPQFI
PDFHFPPFPFLEEYGGEFTAAVITLGLNSTAYVAEIMRAGIQSVDKGQMEAARSLGMKPWMAMRLVVIPQAFRTVLPALG
NELVVVIKETSIVSVIGLNELMFNADTVKGSLSLPFEPLIIAAFIYLVITLTLSKLVGMLERRLTKSNA
>lcl|NZ_CP026520.1_prot_WP_042233220.1_4107 [locus_tag=PC41400_RS20730] [protein=transporter substrate-binding domain-containing protein] [protein_id=WP_042233220.1] [location=complement(4612863..4613678)] [gbkey=CDS]
MKKWGLTLIALVLTMSLIAACGKKTETPKDAAGGAQTTAPAKKLILGTSADYPPYEFHKSTNGKDEIIGFDIMIAKEIAK
DMGAELEIKDMKFDGLLPALQSGNVDMILSGMTPTEERKKNVDFSKTYYTAKQMIVVRAEDKDKYKTMDSLAGKRIVVQK
ASIQETVAKEQIKDAKIKTLSKITDIVLELKNKNADAAIIELPVATSHVNRDKSIVIAEATPKVEEAGSAVAFKKGSQDK
IDLVNKTIDRLLAEKKIDQFVTDASSQVESE
>lcl|NZ_CP026520.1_prot_WP_042233218.1_4108 [locus_tag=PC41400_RS20735] [protein=hemolysin III family protein] [protein_id=WP_042233218.1] [location=4614003..4614638] [gbkey=CDS]
MEYSPREEFANAVSHGIGTLLSIAALVLLVIQATTYGNTWHIVSFSIFGTSLILLYLCSTLVHSAPGGRLKDIFEIMDHS
AIYVLIAGSYTPFMLVTLRGVLGWTLLGIVWGLALIGIVLKIFFVKRFIVLSTVCYIGMGWLIVLAFKPLAEHLSAYGII
WLVAGGLLYTVGTVFYLWRRIPYHHAIWHSFVIGGSVCHFFAVMQVLPVPA
>lcl|NZ_CP026520.1_prot_WP_042233215.1_4109 [locus_tag=PC41400_RS20740] [protein=GDSL-type esterase/lipase family protein] [protein_id=WP_042233215.1] [location=complement(4614678..4615331)] [gbkey=CDS]
MGRALRYIAFGDSLTVGTGDPTREGFTARYRGMAEAALGRSVVLRNAGTNGATSGELLQYLRNEGDLRRGLATSDIVTIT
AGGNDLIRSAMPYLKSRDTGVLKRSLRTFGGNLRQIVRYTQLPVPAGKSPLVILVGLYNPISMLPEAEFWIKRFNQQMER
LQSRTVRYVDVYSAFKGAESRLISDDLFHPNAEGYKRIAECIAQSVPLTSLAGGGGH
>lcl|NZ_CP026520.1_prot_WP_042233213.1_4110 [locus_tag=PC41400_RS20745] [protein=hypothetical protein] [protein_id=WP_042233213.1] [location=complement(4615402..4615713)] [gbkey=CDS]
MNKYGSDNWQDHRFTLACKEVERRGQWLDIYLALHPEEGETLPEELYDPSVYVICNLEGTIVQIILLDEGCDCEYQFTPG
EKEKIARFLEEHCSAEMNSLLLR
>lcl|NZ_CP026520.1_prot_WP_042233212.1_4111 [locus_tag=PC41400_RS20750] [protein=M20 family metallopeptidase] [protein_id=WP_042233212.1] [location=4615954..4617126] [gbkey=CDS]
MKERIYNTIDFHTPEFQAISTYIGQNPELGHEEFLACARLIEELERHDFTVERAVLDIPTAFIGVYDSGKPGPTVAFLAE
YDALPELGHACGHHLICMMSIGAAVGLKSVIDETGGVIRVYGTPAEETKGAKVPMAAAGLFDDVDFALMAHPYYTYERSG
ESLALDAIQFEYFGKAAHAAASPYEGVNALDAVLMLFNSVNALRQQLRSDVRIHGIITEGGQAANIIPDYAVAQFYVRSS
SRAYTNEVAAKVLKCAEGAALQTGCTLTTSNYEYSYDELNTNETLSAVFNRNLSSAGITEDMIQTGKDHGSLDLGNVSLH
CPTIHPYMKVIEEKHLLHTPEFRDLAMKQRALDGMILAAKMLAATAYDVITDPALLAEIKAEFERTRSAR
>lcl|NZ_CP026520.1_prot_WP_042233210.1_4112 [locus_tag=PC41400_RS20755] [protein=amino acid ABC transporter permease] [protein_id=WP_042233210.1] [location=complement(4617256..4617909)] [gbkey=CDS]
MDFSGAYSADNLMFLLKGFLVTLEMAVIAIILSFVIGSLIGVIRYSKIPVLSQLMAVLVEVIRNLPLLLIIFFVYFAVPE
VTGVEMSILTSAIISLTIFESAMIAEIIRGGLNSIDKGQIEASRSSGLTYVQTLWHIVFPQAVRRMVPPIVSQFISLLKD
TSLTVIIAAPELMHNAQIINGSSQKFIVPIFLLAALMYFVVNYFISLFAQRFELKRT
>lcl|NZ_CP026520.1_prot_WP_042233207.1_4113 [locus_tag=PC41400_RS20760] [protein=amino acid ABC transporter permease] [protein_id=WP_042233207.1] [location=complement(4617960..4618616)] [gbkey=CDS]
MLPVDLSIIAENWDRYIEGFGNTVMVSLLSLVLSFLLGTLVAVLRISPFKTLRAISGVYVEFVRNIPLLLIIFFFYYGSS
EFGLSLDGFTAGTIGLTIYTSAFIAEAIRAGILAVPAGQAEAARSSGLSYGQTMIHVVLPQAIKIVIPPLGNQFLNLIKN
SSILGVIAGMDLMYFGDLINSDTLDTFNVYIGVALFYLVLTIPLSLLVQYLERRMARA
>lcl|NZ_CP026520.1_prot_WP_042233205.1_4114 [locus_tag=PC41400_RS20765] [protein=transporter substrate-binding domain-containing protein] [protein_id=WP_042233205.1] [location=complement(4618693..4619544)] [gbkey=CDS]
MKVWKPFKIGMILLTFILVLGACGKTKDNGGSTPGGSAAPTTDTKTLDAIKQRGKFVVGVKFDTNLFGLKDPGTGQVNGF
DIDISKAIAKKLLGDETKVELKEVTSKTRIPMLQKGDIDAIIATMTISEERKKEVDFTDVYFKAGQSLLVKKGSDLKSID
DVVKKKGVKVLTVKGSTSAKNIRAKAPDAQVLEFENYQEAFTALKSGKGEALTTDNAILYGMTKQDPGYEVVGGTFTDEP
YGIAVRKGDAEFTKAVNDVLKELKDSGEYAKIYEKWIGEKPAE
>lcl|NZ_CP026520.1_prot_WP_042233203.1_4115 [locus_tag=PC41400_RS20770] [protein=amino acid ABC transporter ATP-binding protein] [protein_id=WP_042233203.1] [location=complement(4619603..4620331)] [gbkey=CDS]
MINFTNVNKYYGSFHVLQNINLEVQKGEVVVVIGPSGSGKSTMLRCINRLETIQQGELTVEGVRVNDKKTDINKLRREIG
MVFQHFNLYPHKKVIENITLAPVKVLGLSKPEAEKRAMLYLEKVGIPDKAQSFPSQLSGGQQQRVAIARGLAMEPNIMLF
DEPTSALDPEMIGEVLDVMKTLAKEGMTMVVVTHEMGFAREVADRVVFMDQGQIVEEALPKQFFENPREERAKLFLSRLL
AH
>lcl|NZ_CP026520.1_prot_WP_042233200.1_4116 [locus_tag=PC41400_RS20775] [protein=DMT family transporter] [protein_id=WP_042233200.1] [location=complement(4620636..4621622)] [gbkey=CDS]
MTTHSNDKPLFPVPLAFIVGIIGISFSAIFVRWSEAPVAVTAMYRLLLTNLIMLPFLFKYTAEIKAIRWKSVFGLIGSGL
FLALHFLLWMASLSYTTVASSTALLTLEPIFVMIGAYWLFRQKNSWGTVIGMLIAMIGAAMIGWGDFRFSGEALKGDLLS
IVSAAAVAVHMLLGKTMRQSMSAFVYSFFVFLAAAAALGVYNLAQGTPLTGYSQKDWLMFLLLAIVPTMFGHYLFNWLLK
YVKAATVSMSVLGEPLGSTVLAFFLLGEAITGIQIGAGFLLLFGIWLFIRSQGRKGKDEMVATPAVPAPLPEIPEERTGA
SRPSLTDS
>lcl|NZ_CP026520.1_prot_WP_042233296.1_4117 [gene=ytxJ] [locus_tag=PC41400_RS20780] [protein=bacillithiol system redox-active protein YtxJ] [protein_id=WP_042233296.1] [location=complement(4621886..4622227)] [gbkey=CDS]
MANWTEVTTLEQWNEIFDQSAEKPFVMLKHSTTCPVSASALEEYDEYLGKNPNEKVNYYLVKVIETRPVSNQIAEDVGVK
HASPQIIYVKDKFAEWNTSHWSVTSKHISAVLD
>lcl|NZ_CP026520.1_prot_WP_042233198.1_4118 [locus_tag=PC41400_RS20785] [protein=PaaI family thioesterase] [protein_id=WP_042233198.1] [location=complement(4622403..4622816)] [gbkey=CDS]
MEQENTVSPQFQHLMERARNTFWEYLGCEIVEAAADKVVVRLVAQQHHLNLIGIVHGGVTSSMLDNAMGVVLMLARPDDK
IVTTNLNVQFVAPMQPGELYVTAEIVHQSRKMLTAIGRVTDAEGRLGTLGTGTFRVI
>lcl|NZ_CP026520.1_prot_WP_042233196.1_4119 [locus_tag=PC41400_RS20790] [protein=long-chain fatty acid--CoA ligase] [protein_id=WP_042233196.1] [location=complement(4622928..4624616)] [gbkey=CDS]
MDSARPWLHHYESEVASTYEYPRHNLAYFLVQAAGKNPQLPALYFMGKKIKYGQLMEACYRFANGLRQLGVKKGERVAIM
LPNCPQSVIAYYGALLAGAVVVQTNPLYMERELEHQLLDSGAVAIVTLDLLYRRVGAVMPRTHLRHVIVSSLKDYLPFPK
NMLYPIKAKRDGADLSVPWSESVRSFRTLLASSSAIPVMEEVNAAEDVALLQYTGGTTGLSKGVMLTHMNLIANTLQTKH
WFHKAADGQEIYLAALPFFHVFGMTVLMNQAVVRSGMMVLVPKFEVSEILSLISKLRPTVFPGAPTMYIALINHPDIREY
DLSSINVCISGSAPLPLEVQDRFEELTGGRLIEGYGLTEASPVTHANPIWGRRKIGSIGIPFPDTEAKVVNPATGEEAGI
DEIGELIVKGPQVMKGYWNRPHESEAVLKNGWLYTGDMARVDGEGYFSIVDRKKDIIIAGGFNIYPREIEEVLFEHPSVK
EAVVIGIPDPYRGETVKAFVIVREGENLTEEELNTWCRDRLAAFKVPRRYEFRTQLPKTMVGKVLRRRLLEEELEQISAA
KE
>lcl|NZ_CP026520.1_prot_WP_042233194.1_4120 [locus_tag=PC41400_RS20795] [protein=acyl-CoA dehydrogenase family protein] [protein_id=WP_042233194.1] [location=complement(4624647..4626428)] [gbkey=CDS]
MIETRKWTKGGSFVIEDLDVNALVTPEDFTEEQRMMMETTRDFVEGEVLPKDEEIEKLDYELTVKLMRKAGEIGLLGADV
PEEYGGLGLDKISSTVINENLSKASAFALSIGAHVGIGTLPIVFFGTPEQKRKYLPELATGEKIAAYCLTEPTSGSDALG
AKTTAKLSDDGQHYILNGSKLYITNAGFADIFIVYAKVDGKDFTAFIVERDLKGFSVGPEEKKMGIKGSSTCPLYFEDVQ
VPVENLLGEVGRGHIIAFNILNIGRFKLGAGCLGASKEAIELSAKYANTRTQFGQPISSFPLIGKKLAEMNIRTYVTESI
VYRTSGMIDTALQNLDHNSAEAGQQSAKAIAEYALECSINKVFASEALDFVADEGVQIHGGYGFIQEYKIERIYRDSRIN
RIFEGTNEINRLLIPGTLLKKAMKGELDLMGKAQALQSELMQLIPGQTFDETLEQAAHLTAMAKKVFLMAGGSAVQKYGM
NLEKQQEVLSNLADMMIQIYAMETTLVRTRKLIAKSGEEKAKNAIQMTEVFIQEAFAEIERLAKDTLAAMETGDMLRTQL
SVLKKLTRSTPVDAVGLKREIAARVIAAEKYVI
>lcl|NZ_CP026520.1_prot_WP_042233190.1_4121 [locus_tag=PC41400_RS20800] [protein=acetyl-CoA C-acyltransferase] [protein_id=WP_042233190.1] [location=complement(4626671..4627855)] [gbkey=CDS]
MKEAVIVSIVRTPVGKAKKGSLAQTRAEDLGRIVLEEAVGRAPGLRKEDVEDIIIGCAMPEGEQGLNFARTMALYAGFPA
AVPAVTINRFCSSGLQSIAYAAERIMLGHADVIIAGGVESMSHVPMTGFKLSPHPKLAVEMPEVYMGMGHTAENVAQRFG
ISREDQDAFAAESHRKAAAAIAAGRFKEEIVPVETSVRGVDAGGKRWEKPVVLDTDEGVRADTTTGVLAKLRPSFALNGS
VTAGNASQTSDGAAAAVLMSRERAEELGLKPLAAFRSFALAGVAPEIMGVGPVEAIPKALKQAGITRDQVDLYEINEAFA
SQCVHIIRELKLDEAKVNVNGGAIALGHPLGCTGTKLTATLVHELRRRGGGYGVVSMCIGGGMGAAGVFEVYGE
>lcl|NZ_CP026520.1_prot_WP_042233189.1_4122 [locus_tag=PC41400_RS20805] [protein=3-hydroxyacyl-CoA dehydrogenase/enoyl-CoA hydratase family protein] [protein_id=WP_042233189.1] [location=complement(4627883..4630282)] [gbkey=CDS]
MVNQIQKAAVIGSGVMGAAIAAHLAGAGISCLLLDIVPTVLTDEETKAGLTLEQPKVRNRLAAGAIERLKKTNPAPLYSE
AFAARITPGNLEDDLARLAGVDWIIEVVVERLDVKKQLLARIEQVWKPGTIVSSNTSGISIQAMAEDCGPEFKRHFLGTH
FFNPPRYMKLLEIIPGESTDPEILRFMADFCEKRLGKGVVTAKDTPNFIANRIGTYGLLVTLREMLEKGYTVEEVDAVTG
PALGRPKSATFRTLDLVGLDTFVHVARNVYDLVDNEAEKTIFDVPAALTGMVEQGWIGEKKGQGFYKKVKTETGKEIQAL
HLGTMEYAPSRKISSASLEAAKNAKGTAGKIKALLSTPDKYSELAWKVVKPVLLYSAGKLGEIADSIVEIDNAMKWGFNW
DLGPFELWDVIGLERSVKRMEQEGDTVPAWVKEWLAAGHTGFYEKREGTTFYYNRGMYKEADTRPEHISLKALKQQNKTI
LSNSGASLIDIGDGVACLEFHSPNNAIGADILTMVQKSTEEVRRNYRGMVLANEGRHFCVGANVMILLMEAQDGEWDEVD
SIIRLFQNAMMTLKKLEKPVVAAPHSMTLGGGVEACLPADRIIFSPETYFGLVEVGVGVIPAGGGCKEMALRSSRLAAGN
PEVDLQPYINHYFETIGMAKVSGSGYDTKRLGYMGPRDSVVLNGDHRIYTAKQAVLAMDREGYVLPQEEKIRVVGDSGKA
VLQIASYTMKLGGYISDHDRLIANKLAHVLAGGDVPAGSLVSEQYMLDLEREAFLSLCGEPKTQQRMQHMLSKGKPLRN
>lcl|NZ_CP026520.1_prot_WP_042233187.1_4123 [locus_tag=PC41400_RS20810] [protein=heterodisulfide reductase-related iron-sulfur binding cluster] [protein_id=WP_042233187.1] [location=complement(4630328..4632496)] [gbkey=CDS]
MLGAIVQLILFLGVTGFGVYLFAKAVYHRYLYVRLGQPVDWGGQFQERLKDFAVQVFGQTKLLKDKKSGIMHIVIFYGFI
ILQFGALDIIVKGLSGRGLPLPGYEAFGLIQEITVAAILIAMGYAAYRRYGEKLKRLKRGWKPSLVVYFIFSLMVSVLLT
LAFDRIREELEPSAMAPISSVLAAGFGWMPFGAAVVLYHVFWWAHLLILLAFLVYVPQSKHFHLLTAPLNILLKRREPAG
RLRKLDLEDEEAESFGLGKIEDLTQKQMLDLYACVECGRCTNVCPASGTGKMLSPMHMIVKLRDHLTEKGAAVTSKSPWV
PAFAFPGSSVHEMNAEGHELGAWNGSGITDIAPTMNAQKHSWVTTDKKPEEMELVGDVMTEGEIWSCTTCRNCEDQCPVA
NEHVDKIVDLRRHLVLMQGSMPHDGQRAMQNIERQGNPWGISRNDRAKWVQEIDPEGELRVPTVKENPEFDLLFFVGSMG
SYDNRSRKITRSLVRLMNEANVNFAILGNEEKNSGDTPRRLGNEFLFQQLCEENTKIFQKYGVRKIVTACPHTYNTLKNE
YPEFGLEAEVLHHTQLLDELVRNGKLTPRHKVNERITYHDSCYLGRYNNVYDQPRSILKAIPGVELVEMTRTRENGMCCG
AGGGMMWMEETAGKRVNLARTEQALEVNPTVISSACPYCLTMMEDGAKMKEAEETVKTRDIAELLEESVFGAKAPAPFQT
VS
>lcl|NZ_CP026520.1_prot_WP_042233185.1_4124 [locus_tag=PC41400_RS20815] [protein=TetR/AcrR family transcriptional regulator] [protein_id=WP_042233185.1] [location=4632768..4633352] [gbkey=CDS]
MTSKKKEKYDLILEAALKVFAEHGFHRSQVSKIAKAAGVADGTIYLYFKRKEDILISLFREKLGELVGKFHMHIEQTTGV
KEALRKVCEIHYTELENNVDLAFVTQIELRQSSLELRKEIGKAVKPYIELIEKLLVQGVEENLFRADLDVKLTRLLLFGA
MDEVVTSWLISGRKYSLSAQVDGTVQFFLQGIQA
>lcl|NZ_CP026520.1_prot_WP_042233183.1_4125 [locus_tag=PC41400_RS20820] [protein=electron transfer flavoprotein subunit beta/FixA family protein] [protein_id=WP_042233183.1] [location=4633407..4634168] [gbkey=CDS]
MDIIVLLKQTFDTEEKIVLENGQISEDGVKFVINPYDEYAVEQAIQLRDEQGGKVVVVSAGPGRSAEALRSALAMGADEA
VLINDERIGGDEHAVAQALHAFLSRETYDLLLGGNFSVDNGAGQVAIRLATLLGIPHVSSITKLEIAGGKATAHRDAEGD
LEVAEVTLPALFTAQQGLNEPRYPSLQGIMKAKKKPFREITLANIGLDTVESRTERVDLFLPPPRKAGKTLKGEPAQTAA
ELVQLLRTEAKAI
>lcl|NZ_CP026520.1_prot_WP_042233181.1_4126 [locus_tag=PC41400_RS20825] [protein=electron transfer flavoprotein subunit alpha/FixB family protein] [protein_id=WP_042233181.1] [location=4634244..4635215] [gbkey=CDS]
MSNTYLIVAEAKGGALRQVSFETIGAAKAIAAEGDSVLAALIGSGVSGLAGELAAYGAQKVYVTDHADLKDYNPEAYGKA
LHDIIRQAEPDAVLLGHTAIGKDLAPQIASLLGAGQISDVTAVESDGGKAVFTRPIYAGKALEKKAFTSFPWVVTVRPNN
IAAPAAAEGASAAVENVAYEAPSLRTVIREVVQKTSGQIDLAEAKVVISGGRGVKSADGFKPLEELAGVLGGAVGASRGA
CDAGYCDYAMQIGQTGKVVTPEIYIACGISGAIQHLAGMSGSRVIIAINKDPEAPIFKVADYGIVGDLFDVVPLLTEEFR
KAL
>lcl|NZ_CP026520.1_prot_WP_042233179.1_4127 [locus_tag=PC41400_RS20830] [protein=anti-sigma factor] [protein_id=WP_042233179.1] [location=complement(4635541..4636482)] [gbkey=CDS]
MSRQESPNMCEWLELYMLGGLSQSEREQMEEHLSHCEACKKEREELQAVTDLLPYTAEPVAVPDGMKQRVLGRILGSEAD
HSGQPPLTGAGALQEQIQAQKDYRVTDASSAAELQKHAGTAQPEPVPAGEHETVSARRPAGKRSRTFIRRAAGLGIAAAA
VILGWVNMQLREDVRSLKGLNASLQQSVSSLQGQLTLAGQPSQASKANKVVKLNPAVEGIVAQGLATIVIDAKGTHLVVQ
AENLPALRDSQAFQVWLIKGEQKVNAGTFLPQGGSGALYYTFEPQDYDTVAITLEPDAGGTQPRGSLVLAAKI
>lcl|NZ_CP026520.1_prot_WP_042233177.1_4128 [locus_tag=PC41400_RS20835] [protein=sigma-70 family RNA polymerase sigma factor] [protein_id=WP_042233177.1] [location=complement(4636479..4637072)] [gbkey=CDS]
MNEWPEDSQLVRRIALKQPQALEQLYDRYERPMYSFAYRIVKDSMMAEEVVQELFLRIWNHAERYEDSQDEDSQGKLSTW
MFAIARNIAIDTLRRKQTRTSGQLAEPDKLLLVADRSRGTEEEVELRFMGEELASAMEDLNEDQKKVIDLIYFKGYTHQE
VSQKQEIPLGTVKSRVRLAMKQLKEKLAGRGKEGAHI
>lcl|NZ_CP026520.1_prot_WP_042233175.1_4129 [locus_tag=PC41400_RS20840] [protein=hypothetical protein] [protein_id=WP_042233175.1] [location=complement(4637341..4637847)] [gbkey=CDS]
MLPSEKRKVAEKRWNRMFSSPRLEQFMKSRAYRTAVPVILLIGLAEHAYWIYHLGKNEPAGFVSRQEWASVLLLTVILFS
GCLRYMGYKRAEGFAESLKTLLCMLLFVLYAFYCGAGCSSAAEAGGGLGGGQSYLLEARTAAFAAAAGTALYAIWKWLGA
KLTRERRE
>lcl|NZ_CP026520.1_prot_WP_042233173.1_4130 [locus_tag=PC41400_RS20845] [protein=MurR/RpiR family transcriptional regulator] [protein_id=WP_042233173.1] [location=complement(4638120..4638980)] [gbkey=CDS]
MSKIADLNNSLLLLNSIYSSLTKSEQKVADTVLKDPEATVFYTITDLAEKAEVGETSVIRLCRKLGYKGYQEFKLSLVQV
LATPSEQVHGKIEESDDLDQIMKKMTSQNSQAVQNSTSLLNVQELQQAIDAILAARKILFFGVGSSGITAQDAKYRFMRL
GFNVDAGSDGHIIAMNASLVEKGDVVVGISSSGSTKDLVDAVRLAKQKGAYIICLTNHARSPIANQSNVVLLATAKETPL
QGGAFSSKLAQIHLLDILSTAVAMQMKDQTYRALEHTAKSVLDKLY
>lcl|NZ_CP026520.1_prot_WP_042233171.1_4131 [locus_tag=PC41400_RS20850] [protein=FAD-dependent oxidoreductase] [protein_id=WP_042233171.1] [location=4639590..4641161] [gbkey=CDS]
MNRELKADVVIIGGGVGGCAAALASAKMGLKVIMTEETHWIGGQLTSQAVPPDEHKWIEQFGCTRSFRQFRNGVRSYYRD
YFPLTAEARAVVNLNPGNGGVSRLCAEPRAMLAVLEQMLAPYVNSGKLTILTRHRITAAETDGDKIRAVKVKHLDLGHKH
TLTAPYFLDATECGDVLPLAKVEYVTGAESRKDTGEPSAPEVANPKDIQAFTVCFAMDYFEGENHVIDKPKDYNFWRNYK
ADFWPDKQLSWISPNPVTLEPAEDTLFPGTDKYPLFNYRRVIDKANFMPGTFPSDISLVNWPQNDYWLGNIYDVPEAEAK
KNLEQAKQLSYSLLYWLQTEAPRPDGKAGYPGLRLRPDVVGTEDGMAQAPYIRESRRIKAEFTVKEQHVSTAIRGDKGAE
VFPDSVGIGAYRIDLHPSMTGRNYLDVSSLPFQIPLGSLIPVRVNNLLPACKNLGTTHITNGCYRLHPVEWNIGEAAGFL
AGYCVQNGYQPREVHADKTKLEGLQNILSWNGVELAWPTITPL
>lcl|NZ_CP026520.1_prot_WP_042233169.1_4132 [locus_tag=PC41400_RS20855] [protein=metallophosphoesterase] [protein_id=WP_042233169.1] [location=complement(4641290..4642342)] [gbkey=CDS]
MKFTFAVLSDIHVVAWDPESHRRLEVCLKDYASMKHPPEFIVINGDLTEGFAEDYTVLKRILSEHVRVPVYVTNGNHEYY
CMWYNDGGFSVDTFPNGWSTAEAMARFLGFLGKPLPYYDLRLMGLRFVFMSAERYRDTDPAIREDAWISDEQFAWLERIL
REESHPRHAAAQSRAETASASEADPAALEAAGPVRGEAGNAAFPGEEEAGLQPADSTGEEAYRPHGRQRPVFVFLHQPLP
HTVAGSYDPAEKGVVQDVRLRELLGRHPNVCFFSGHTHYTLVSPGIYFEDGCSHMGTSSVRRPIGDGIDPAAPVESESVL
VEVAPHSVTIRGRRHNTGEWLSLARFVKRY
>lcl|NZ_CP026520.1_prot_WP_042233166.1_4133 [locus_tag=PC41400_RS20860] [protein=family 10 glycosylhydrolase] [protein_id=WP_042233166.1] [location=complement(4642492..4645500)] [gbkey=CDS]
MTVKRSLRSSGLLRRTLFAVLLLSVAAAGCLPLYGIAGAAPGSAATEGYPQHSGPYGGIAVPPDSPKSGPADQDPFRSAN
KPGTPLLPHTSLGVTVNESVYGPHIKLPDGTLHPVAGMDQERKEDALTVYTSTYGTYTPPFDDRTIELIVTEGIIVDKIS
DGKKGTYLPPNGYVLSGTASTKALLQQLNVGDSLTAVNIKIPVLPTKYVKVGDLIIPVDRVNGPRNQGEAVLFDSGYGAS
TRTNPWGIEVIFQNGRVTGVKGILMDSGGKFIDNDSPIPKGGYVLSVQAASPYYADLLALAAPGLAVSMELNAYPMYFAS
KMAYDAFNPRTREDNPAGWDDTNNKPYPGHRGAEQLIVYDSSYGAATGTNPWGGEAAVNADGKIISSGGNNRPIPEGGYV
LSGHGSKMAWLSRYAAIGATVKINRTEKQVLLIFTPESYLDKARLTAASAEQAMEESKRRFMDVPYEEISKEIQQVKQLE
IQAGKHLEEKGVEGLTEVLAELDGKATRVHYMNVESRTVENRGIWIRPKEKSLEEVKRALDKIKAAHLNSVYLETFWNGY
TIYPTKNNITAHNPIYGGFDVLQAYLDEGKKRGIEIHAWVENFFVGGPEPGPIVERKPEWSLLSRKGQNYQEDATSGFRY
YFINPALDETHDFLIGVYKELLSKYDVAGLHLDYARYPGSGDFTNDFGYDAYTRSLFQKKYGTDPLLLHPGDALWAQWVD
MRVGFVDKFVFRLAKEVKKWKPQVKLTAAVWPEYEIARGDVLQDSMNWVKHNLIDNVFPMSYKLDTSLVVADARKTFEIA
KDKSYVTIGVGSFIDLTKDVFIDQIISSHGLGISGSAVFEYESFFDQGYDKNTVLGVYRNEAVVPDSDPARSIAAILAGQ
KRKVKELYLPFKGMDVRKAARYTDEINEIVKKLGGEKNPRLNPGRLKQVKKSLASLVKDIGSDSRLNAEVRERMLYDLKG
ALRIAEITAAKSGKGGKNADEADDFNETGVTHAEDEAGEANE
>lcl|NZ_CP026520.1_prot_WP_241688713.1_4134 [locus_tag=PC41400_RS20865] [protein=tryptophan-rich sensory protein] [protein_id=WP_241688713.1] [location=complement(4645768..4646550)] [gbkey=CDS]
MHALRKMMNIVGLLLVLIVNTLAQVLPINGVTTGEISDRHHTYVTPAGYVFTIWSVIYLGLIGFVIYQARKDTGNRPAVE
RISYWFVVSCLLNSAWIFLWHYEKIGWSTLVMAALLLTLIAIYQRIRYGRESRAIGEYSFVQVPFSLYLGWICVASIVNV
TVWLQDLGWNGWGIDPRMWAVILLTAAGVLAVAFSYGKKDVPFALVFVWAFIGIGIEQQADTWVRNAAWLLAAIVGAATL
WLLAKVIRDRKRWDQSYISR
>lcl|NZ_CP026520.1_prot_WP_042233161.1_4135 [locus_tag=PC41400_RS20870] [protein=Xaa-Pro peptidase family protein] [protein_id=WP_042233161.1] [location=complement(4646687..4647778)] [gbkey=CDS]
MNNRLKQISDYIEQNNLDAVIITVPRHVYYLTGFASDPHERFLGLILPRGEEPFLLVPALDAEAAAQASTVQRIHTHSDT
DSAYEVLRGLLPASLGSIGIETEHWTVSRFEELKSCIGVSRFSSVDELLRDMRVIKSEDEIGLMKDAVTLVEEVLRRGLQ
KVAVGVTEIELVAELEFQMKKLGAQSPSFSTIVLTGPASALPHGVPGTRKISRGDLLLFDLGVYLNGYASDITRTFAVGE
VSEEQRRIYNAVLAGNEAAIQAVRPDVTCASVDAAARNRIEAGGYGTYFNHRLGHGLGIDVHEYPSVHGRNEALLKAGMT
FTIEPGVYVPGVGGVRIEDDVLVTAGGVEVLTSFPKQLTVLGD
>lcl|NZ_CP026520.1_prot_WP_042233159.1_4136 [locus_tag=PC41400_RS20875] [protein=FAD-dependent oxidoreductase] [protein_id=WP_042233159.1] [location=4648038..4648601] [gbkey=CDS]
MYDLVIIGAGPAGASAGLIAARAGKKTLLIDSGQSMTKRAWIENHYGVAEISGPDMVELGKKQAAKFGAEIAEGKVESVD
KQNDGFLVTSDQGSYEARHVLLATGVLADLADKIGVTTVPGTEPRIKTIVQTDANNKTSVEGIWAAGTIAGMSVHTIVTA
GDGAKAAINIVSELNGERYVDHDVLKS
>lcl|NZ_CP026520.1_prot_WP_042233157.1_4137 [locus_tag=PC41400_RS20880] [protein=rhomboid family intramembrane serine protease] [protein_id=WP_042233157.1] [location=complement(4648751..4649365)] [gbkey=CDS]
MLFVRQESFKEYIRRYPVTSLLVLIHVVLFVLMEVNGSSTSTRTLIEYGALFSAPGFAPEWWRFFTAMFLHIGLSHLLFN
SFALVIFAPPLERLLGSVRYAVFYLASGALGSAFSYWLHTDSYVAAGASGAIYGIYAAYLYLALFRRQLLDQQSRQTVII
ILVSGLLFSVIVPNVDLYTHLGGFLAGFILFALFVEFIKRSGRR
>lcl|NZ_CP026520.1_prot_WP_042233155.1_4138 [locus_tag=PC41400_RS20885] [protein=type 1 glutamine amidotransferase domain-containing protein] [protein_id=WP_042233155.1] [location=complement(4649472..4649978)] [gbkey=CDS]
MSKIAFLLAPQFEDSEMQEPYEALKTDGHEVVIIGLKQGELLTGKNKKAEYTADQGIKEAKADEYDAVVIPGGSSPEGLR
LDADIQSFVKKIHEAGKPVAAICHGPQILISAGLIKGRKLTSYPPLQDDVRNAGGEFADQEVVIDGNLITSRTPKDEPAF
IAAIKKAI
>lcl|NZ_CP026520.1_prot_WP_042233152.1_4139 [locus_tag=PC41400_RS20890] [protein=hypothetical protein] [protein_id=WP_042233152.1] [location=complement(4650010..4650444)] [gbkey=CDS]
MSKHIQAYFRTENDAEDVRIRLQTYSVEQLEVGETQGRVSRTGLIAPPAFAGGTSGAVGVPGATIGTSPSPGGFIPVRAA
EDRHDPDRTIRGEDDGPLSDFILGEGDDDSENLRYVLSAKVPDSDYTEIVEIIRSNKGYVESID
>lcl|NZ_CP026520.1_prot_WP_042233149.1_4140 [locus_tag=PC41400_RS20895] [protein=LCP family protein] [protein_id=WP_042233149.1] [location=complement(4650599..4652002)] [gbkey=CDS]
MSDPINPNSTPVSGTDSRKSKLKKKAKWPLWKKIAVTGGIIVLLLVIGIGTYASFLISKANNIIDTVGTKEVVAPKDSAK
VKPISILLMGVDSRKETGSLNSDVMMAISMNPETKSATIVSLPRDLEMKLKKNGDSHKANYFYPHFTLTEPETAMKKTKD
FFGNFFEVPFDYMVTIDFEGFKQVVDQLGGVTVDVDQDMYYVDNWDGTRIDLKKGVQKLDGKKTLDFVRYRKSNPGSKFE
TQESSDPARNARQQQVLNEILKNLKSLGGVTKASGILEAVGNNLKTDIPASQIHDMISQYITIDTQNIELIHMDGDWISP
YIVVKDEEIREAKNALKLRLDPNWVPSSSPNSTDSSKGTDSRGNNTNSDGKITPTRTPKPTDAPTHTPKSTIKPTPTPTP
KTSVRPNPTPEDDDVVPGLTPDPTRKPGGSVTPPPSGKPTTPPKNTELPKPTPTPTPQNQEAAQKTG
>lcl|NZ_CP026520.1_prot_WP_009672820.1_4141 [locus_tag=PC41400_RS20900] [protein=universal stress protein] [protein_id=WP_009672820.1] [location=4652193..4652627] [gbkey=CDS]
MQFLNILVAYDGSEVSKKALEKAVGLAKSNPSAKLEVVHVANMPNLVVGEALISTPAGMSGEYYELAEQIKDDAKQRLVS
LSQPAEVYLLNGNPGRAILEHAERTGRDLIVIGSRGLSGVREWVLGSVSHYVVQHAQIPVLVIK
>lcl|NZ_CP026520.1_prot_WP_042233145.1_4142 [locus_tag=PC41400_RS20905] [protein=glycosyltransferase family 39 protein] [protein_id=WP_042233145.1] [location=complement(4652718..4654145)] [gbkey=CDS]
MEHKTRIPAWLITALLFIAIAALRIPYITNSPYEIDSWRQSDTESIARHFVETKFNLFYPQFNYQGPLPNYVELEFQITP
FLIAILYKLFGYHYWLARLVPLAFFMGSAWFLYLIARTFYTKRTAWIAVLLYGILPVAVTYSRAIMPEAASLFFYLGAFY
AFTVWIRKGGTAFVLTAALFTALAISLKVPTIFVGIPMLFMAAAKFKWKMLVNWRLWLFAALALGLPALYYAWASSIAEF
QFVSGIATKHIFPKMATALFTPETWDFFLRELPKAFSWPLLALAAAGLLAGFRQKDIPIRVWGWAMLLELATIVAVIKFD
YYLIFFAPVAALLAAKTIGRLTEFRFGRISAAVVVAGIGWACALMMNPDYGKEKTVILEQARAVERLTKPGDLIVVGTDD
PSLLNASHRVGWRVLNSIPDDPIKELEYFRAGGAAYFVPLKGQIDGDKAGELRRYLNAKFEKIPVEGGYDIYRLN
>lcl|NZ_CP026520.1_prot_WP_042233142.1_4143 [locus_tag=PC41400_RS20910] [protein=EamA family transporter] [protein_id=WP_042233142.1] [location=complement(4654156..4654518)] [gbkey=CDS]
MSALWVILLSVGMGAVGQVLLKAGANRLGELSLAPATLIPDLFRMVKTPEILIGLILFGTSFLLWVKVLTKVELSYAYPM
MSLSYVIVFVMSFLWFQETFTMQKLVGMAVIIIGIIIINK
>lcl|NZ_CP026520.1_prot_WP_042233139.1_4144 [locus_tag=PC41400_RS20915] [protein=glycosyltransferase family 2 protein] [protein_id=WP_042233139.1] [location=complement(4654515..4655297)] [gbkey=CDS]
MKETIIVGLPAYNEQDALPKLLPKMEQLSVQFGEAFRVVVVNDGSSDGTAEVLKEACRTYAWLQVIDHERNRGLGEAMNT
LVRFVTSEYQSSDVLVTLDADNTHNPNLIPALVAKLKENKLDLVVASRFARGGQELGLSFIRRLYSRGAMVFFKLFFPIE
GVHDYSSGFRAYRIGYLNQAMNLYKQRLITTSGFDCMAEMMARFSKIGIKAGEYPLVLEYHLKVGRSKMNVKRTIAGYFR
LLMKVKPPVQGTQKKVKGTV
>lcl|NZ_CP026520.1_prot_WP_042233137.1_4145 [locus_tag=PC41400_RS20920] [protein=four-helix bundle copper-binding protein] [protein_id=WP_042233137.1] [location=complement(4655667..4656020)] [gbkey=CDS]
MPHDMTHEGVNEEIQRCIEECINCIKTCNYCFDACLKEEDASHMAACIRLTRECADICTYAASAMTRQSPFIVEICNLCA
TVCEICAKECAEHEHEHCRECAKVCRSCAEACRRMAA
>lcl|NZ_CP026520.1_prot_WP_042233135.1_4146 [locus_tag=PC41400_RS20925] [protein=DUF2087 domain-containing protein] [protein_id=WP_042233135.1] [location=complement(4656109..4656879)] [gbkey=CDS]
MKDVSELFWNASTDEIKRGYVFSDQEEAYVCLVCGKSFTKGIIYREDETFYEAEKFASRHIAAEHGSMFHYLLNLNKKLT
GLTDLQKQLLGFFHDGLSDKDIVGELGGGSTSTIRNHRFTMREREKQAKVYLALMGLLDQRPDTGKTSSFIAVPKSATVL
DERFAITEQENEQFLKLYFPYGVDGPLNKFPKKEKRKIAILRQLATRFEADRKYTEKEVNEVLKEAWHDYVTIRRYLIEY
GFMKRHDDGSAYWVTR
>lcl|NZ_CP026520.1_prot_WP_009674934.1_4147 [locus_tag=PC41400_RS20930] [protein=YlbF family regulator] [protein_id=WP_009674934.1] [location=complement(4657021..4657437)] [gbkey=CDS]
MEHYTNTEMIVREDILAKTKELAELLTTSSEVQFFQQAEKQINQNEHIQSLIQAMKKKQKEIVAFEKFDNEIMVKKIEKE
IEDLQDELDNIPIVSEFQQSQADINYLLQLVMGVIRDTIADKITVESGSAQPPSSCSE
>lcl|NZ_CP026520.1_prot_WP_042233133.1_4148 [gene=miaB] [locus_tag=PC41400_RS20935] [protein=tRNA (N6-isopentenyl adenosine(37)-C2)-methylthiotransferase MiaB] [protein_id=WP_042233133.1] [location=complement(4657470..4658954)] [gbkey=CDS]
MTVKTAKSDKDFSIYFQPPSYKDAKKRGKENIEVLYDFNIPEEMKEFGQGKRYMIRTYGCQMNEHDTETMKGMLEELGYT
STEEKQEADVILLNTCAIRENAEDKVFGELGHLKHLKLEKPNLVLGVCGCMSQEEGVVKRILQKHGHVDLIFGTHNIHRL
PALLQDAYLSKEMVVEVWSKEGDIIENLPKKREGMRAWVNIMYGCDKFCTYCIVPYTRGKERSRRPEDVLAEVRDLARQG
FKEITLLGQNVNAYGKDFEDISYSFGDLMADMRKIDVPRIRFTTSHPRDFDDHLVEVLSHKGNLMEHIHLPVQSGSTEVL
KRMSRKYSRELYLDLVRKIKTAIPGAVLSTDIIVGFPGETDEQFEETLSLVKEVRFDFAYSFIYSPREGTPAAAMEDNVP
EDVKKERLYRLNEVLGAFSKESNEKLRGEVVEVLVEGESRKNSEVLSGRTRTNKLVLFKGSKDLIGSFAHVRVTEPQTWL
VKGELVAEPLQMLS
>lcl|NZ_CP026520.1_prot_WP_042233131.1_4149 [locus_tag=PC41400_RS20940] [protein=phosphate propanoyltransferase] [protein_id=WP_042233131.1] [location=complement(4659110..4659679)] [gbkey=CDS]
MKQIPIGISARHIHLKPEHITRLFGEGYELTVLKDLSQPGQFAAGETVAVVGPKGRFDKVRILGPARNISQLEISRTDAF
ALGVNPPVRESGNIEGTPGIKIVGPKGEVELEEGVIVAARHIHFHTEDAKKWGIEDKQRLRVRVETDRPVIFEDVIARVS
DQFALDMHIDTDEGNAAGLKTGDVGIIVD
>lcl|NZ_CP026520.1_prot_WP_042233129.1_4150 [locus_tag=PC41400_RS20945] [protein=2-oxoacid:ferredoxin oxidoreductase subunit beta] [protein_id=WP_042233129.1] [location=complement(4659801..4660667)] [gbkey=CDS]
MATLKDFRNNVKPNWCPGCGDFSVQASIQRAAANVGLEPEQLAIVSGIGCSGRISGYVNCYGFHGIHGRSLPIAQGLKMA
NRELTVLASGGDGDGFAIGMAHTVHAIRRNIDMTYIVMDNQIYGLTKGQTSPRSGEGFKTKSTPSGSIETPLAPLEMALA
AGATFVAQSFSSNLKQLTHIIEEGIKHQGFSLINVFSPCVTFNKVNTYDWFKENIVDLETIPDYDPSNRIQAMTKLMETD
GMLTGIIYQNKTKPSYENLVTGFKAEGLAKQDMKLSQADFDKLVAEFK
>lcl|NZ_CP026520.1_prot_WP_042233127.1_4151 [locus_tag=PC41400_RS20950] [protein=2-oxoacid:acceptor oxidoreductase subunit alpha] [protein_id=WP_042233127.1] [location=complement(4660655..4662406)] [gbkey=CDS]
MISQLSWKIGGQQGEGVESTDRIFSTALNRLGYYLYGYRHFSSRIKGGHTNNKIRISTKPIRSISDDLDILVAFDQETID
LNSHELGVHGVIVADAKFNPTLPEGVNARLFPVPITAIAEELGTSLFKNMAASGASWALLGLPLDVFNKAVEEEFGRKGA
AVVEKNIEAVRKGAESVLELAGGPIAELQLEPADGEQKLFMIGNDAIALGAVAAGCRFMPAYPITPASEIMEYLTKVLPK
FGGAVIQTEDEISACTMAIGASYGGVRAMTASAGPGLSLMMEAIGLAGMTETPVVIVDTQRGGPSTGLPTKQEQSDVNAM
IYGTHGEIPKVVLAPSTIEECFYDTIEAFNIAEKLQLPVILLTDLQLSLGKQSSEVLDYNKIVIERGSLIGQPEEVEDSK
QLFKRYEFTENGVSPRVLPGVKYGLHHVTGVEHDQEGRPSESPVNRKKMMDKRLEKLKHVEVTNPILVNAPHEEPDVLII
GMSSTGGTIEEARGRLEKDGLATNHIHVRLLHPFPAEELKPHLEKARNVIVVENNATAQLANLIKLNAGSADKIINMLKY
DGTPFLPSEIYQECKELTVAWQR
>lcl|NZ_CP026520.1_prot_WP_042233125.1_4152 [locus_tag=PC41400_RS20955] [protein=dipeptidase] [protein_id=WP_042233125.1] [location=complement(4662630..4663568)] [gbkey=CDS]
MIIDAHCDVLMKMYRDPLLDFNAAEPLLDVSYPLMKQGGVKIQFMAVYLPEEIKNPDVGHILEYVDILHRRVLCNPDIVL
IQSRSDLQHVLTTGKRGVLLTLEGAEGLRGSLAQLRMLHRLGLRSVGPAWNFANWAVDGAMETRNGGFTLRGQQLIKESN
QLGLLFDASHLSEKAFWEAADVPGLHLFASHSNAKSACDHPRNLTDGQIRCLIERKGMIGITFVPWFVKKTGSVFITDLF
RHIDHICSLGGEDFIGFGSDFDGIDNYIRGLENAGMYVNLANELYKHYTPVQAQKFLWQNGYAFLERSLPVS
>lcl|NZ_CP026520.1_prot_WP_005550495.1_4153 [locus_tag=PC41400_RS20960] [protein=stage V sporulation protein S] [protein_id=WP_005550495.1] [location=complement(4663687..4663947)] [gbkey=CDS]
MEVLKVSAKSNPNSVAGALAGVLRERGAAELQAIGAGALNQAIKAVAIARGFVAPSGVDLICVPAFTDIVIDGEDRTAIK
LIVEPR
>lcl|NZ_CP026520.1_prot_WP_042233122.1_4154 [locus_tag=PC41400_RS20965] [protein=TIGR00282 family metallophosphoesterase] [protein_id=WP_042233122.1] [location=complement(4664100..4664894)] [gbkey=CDS]
MKIVFIGDIVGSVGRNAVKTMVPKLKLKYANAVFIANGENAAGGRGITSQITKELYQLGIHGITLGNHTWDQKEIFDFID
DEVQMVRPANFPEGAPGRGHTVLKVRNQEILLINLQGRTFLPPIDCPFRKVDEILEAAKGKYKHIFVDFHAETTSEKIAM
GWHLDGRVSAVVGTHTHVQTNDNRVLPQGTAYVTDVGMVGPRDGVLGMERTAVLRKFTTQLPVRFVVDEGQWHFHATVID
LDDTTGKAGKIELIRINEEDLLFE
>lcl|NZ_CP026520.1_prot_WP_036739444.1_4155 [gene=rny] [locus_tag=PC41400_RS20970] [protein=ribonuclease Y] [protein_id=WP_036739444.1] [location=complement(4664979..4666520)] [gbkey=CDS]
MNPIIWVLIILVVGVICLGIGYFIRKSLAEAKISSAETAAAQIVENARKEAEALKKETVLEAKDEVHKLRAEAEKDIRER
RNESQRHERRLLQKEESLDKKLESLERKEEQVANKEKRIEETQNQIDELYRKQVSELERISGMTMEEAKTIILSGVEQEV
RHETAQLIKEIEQQAKEEGDKKAREIITLAIQRCAADHVAETTVSVVSLPNEEMKGRIIGREGRNIRALETLTGIDLIID
DTPEAVILSGFDPIRREIARTSLEKLVADGRIHPARIEEMVEKSRKEVDERIREYGEQATFEVGVHGLHPDLIKILGRLK
FRTSYGQNVLKHSMEVAYLSGLMAGELGEDVTLAKRAGLLHDIGKALDHEVEGSHVEIGVEIAKKYKEHPVVVNSIASHH
GDVEATSVIAMLVGAADALSAARPGARRETLETYIKRLEKLEGISESFEGVEKSYAIQAGREVRVMVQPEKIDDAEAFRL
ARDITKKIENELDYPGHIKVTVIRETRAVEYAK
>lcl|NZ_CP026520.1_prot_WP_042233120.1_4156 [locus_tag=PC41400_RS20975] [protein=RecX family transcriptional regulator] [protein_id=WP_042233120.1] [location=complement(4666836..4667495)] [gbkey=CDS]
MKDETETSVITGIERQKRNKKRYNVYVNDEYACSVHEDILIKNRLLKGEQIDAQRLRQALEEDEKHEAYLKAIRFVAVRP
RSVQEVKMKLKDLGYEADLIDWTVGRLLEQKYLDDADFAKLWTEQRIQYQRKGRNAVRYELAQKGLSRDEISKALSAVDT
DEEMEGALIVGRKKWKTVSGEKMDRKRKTMAFLLRRGYPSSLVNDVMRRLSEEELENEE
>lcl|NZ_CP026520.1_prot_WP_042233118.1_4157 [gene=recA] [locus_tag=PC41400_RS20980] [protein=recombinase RecA] [protein_id=WP_042233118.1] [location=complement(4667605..4668666)] [gbkey=CDS]
MSDRRAALEMALRQIEKQFGKGSIMKLGESTHMQVETIPSGSLALDIALGTGGFPRGRIIEIYGPESSGKTTVALHAIAE
VQKSGGQAAFIDAEHALDPLYASKLGINIDELLLSQPDTGEQALEIAEALVRSGAVDIIVIDSVAALVPKAEIEGEMGDS
HVGLQARLMSQALRKLSGAINKSKVIAIFINQLREKVGVMFGNPETTPGGRALKFYSTVRLDVRRVETIKQGNDMVGNRT
RVKVVKNKVAPPFKQADIDIMYGEGISREGSIIDIGTELDIVNKSGAWYSYNGDRLGQGRENSKQFLKENKEIADVIEQK
IREASNLTNVVPPAGVDVDDPDEELALFTEEEE
>lcl|NZ_CP026520.1_prot_WP_042233116.1_4158 [locus_tag=PC41400_RS20985] [protein=competence/damage-inducible protein A] [protein_id=WP_042233116.1] [location=complement(4668955..4670208)] [gbkey=CDS]
MKAELIAVGTELLLGQILNTNAQYLAQECAMLGIDLYFQTVVGDNPARIREALEQAGKRADLVITTGGLGPTLDDLTKDV
LADMTGRRLYIDEISMAKIRELFDSQGLHMVESNSRQANMIEGAEPLPNDTGLAVGLGLDHEGTRYLLLPGPPKEMKPMF
ERYAKDWLSGLMTDEMPLYSKMLKFAGIGESSLEHKLFDLIEKQTDPTIAPYAKEGEVTIRLATKAHTEEEALRKMEQVE
KEIRSRVGEHLYAEEDIPIEEAVLAILQSRNQTLTTAESCSGGLLSDLITSIPGRSSVFLGGVVSYSNEMKHSMLGVPTE
LLEGEDAPGAVSAETAEAMVRGLLERTNGDYAVSVTGVAGPTTSEGKPIGLVYVGVGCKDGDVHVQKLQLPGLRETIKLR
AAKTALHQLWKLIKADG
>lcl|NZ_CP026520.1_prot_WP_009674945.1_4159 [gene=pgsA] [locus_tag=PC41400_RS20990] [protein=CDP-diacylglycerol--glycerol-3-phosphate 3-phosphatidyltransferase] [protein_id=WP_009674945.1] [location=complement(4670454..4671041)] [gbkey=CDS]
MNLANKITLARIFLVPIIMFFLLVNIQFPHIKIQTFSITYNQIIAALIFIIAASTDSLDGYIARKNKLVTNLGKLLDPLA
DKLLVTAVLVSLVEMDKLGAWIVIVIISREWAVTGLRQIALLEGAVIQASKWGKAKTAAQITAIIALLINNFPFAFINVP
FDLIASWIAAIITVYSGLDYFIKNKHVIDFSEKTQ
>lcl|NZ_CP026520.1_prot_WP_042233114.1_4160 [gene=rimO] [locus_tag=PC41400_RS20995] [protein=30S ribosomal protein S12 methylthiotransferase RimO] [protein_id=WP_042233114.1] [location=complement(4671041..4672369)] [gbkey=CDS]
MTEKVRVVTLGCEKNLVDSEIMSGLVDQRGYSVVENNEEATVIIVNTCGFIDAAKEESVTTILDMADLKRTGNLKALIVS
GCLTQRYKEELMKEMPEIDGIVGTGDFHNINAIIDEALKGKKPVYVGNPVFNYEQALPRKVSTPRYTAYVKIAEGCDNAC
TFCSIPIMRGKFRSRSMESVISEARQLASQGVKEVSLIAQDSTNYGTDLYDKFMLPELLNRVSEVEGIEWVRLHYAYPGF
FTDELIETMAANPKICKYIDMPLQHSSDAILKRMRRPGREKDARELILKIRKRMPDAALRTSLIVGFPGETEEDFEHLVQ
FVQDVQFDRLGVFTYSMEENTPASRLPGQVPDEVKEYRAAALMELQREISNRRGGRRLGEAVDVLIERYDGRNDVYIGRS
QYDAPEIDGEVFVTGTNIGIGGVVKVRVTHSFEFDLSGEVIA
>lcl|NZ_CP026520.1_prot_WP_042233112.1_4161 [locus_tag=PC41400_RS21000] [protein=YajQ family cyclic di-GMP-binding protein] [protein_id=WP_042233112.1] [location=complement(4672497..4672988)] [gbkey=CDS]
MSSESSFDIVSKLDMQELNNAITQAKREIETRFDFKGSKSEITLEKEELVLLSDDDFKLQNVIDILHAKMAKRGIALKNL
EYGKIESASLGTVRQRVKLKQGIEADIAKKINIMIRDSKLKVKSQIQGDSIRVTGKSRDDLQKVIQMLKGADLPIDLQFD
NFR
>lcl|NZ_CP026520.1_prot_WP_042233110.1_4162 [locus_tag=PC41400_RS21005] [protein=DUF4115 domain-containing protein] [protein_id=WP_042233110.1] [location=complement(4673034..4673918)] [gbkey=CDS]
MSELGQRLKQARLEKRMSLDDLQEATKIRTRYLEAIEEENYKVLPGSFYVRAFIKSYAEAVGLDPNEVLNMYQGTVPAAP
PEPTVETIRTPRRNSARSTDRLSKWASTIMVVSFVLLIAAIVYYFVVVNNKGGSSGAINEPNRLTDTADPNATSVPSPAP
GSGGGATATTTPTPTPTPSATPTPTPTPGGALTLVNSSNSLDNYTAVGGKLDITVTAAGTQCWYRVDSFPNNSKKMEKQG
TLKQGETETFSLTTTAQIRVGAANAAKITVNGQELNVGDASNPKNIKITLQPQP
>lcl|NZ_CP026520.1_prot_WP_042233108.1_4163 [locus_tag=PC41400_RS21010] [protein=DUF3388 domain-containing protein] [protein_id=WP_042233108.1] [location=complement(4673972..4674736)] [gbkey=CDS]
MEIKEWYMEYRIHKNRPGLLGDIASLMGMLDINIVTINGVEDRTRGMLLQTNDDEKIELMGKMLKKVDNITITAFRTPKL
VDILAVRHGRYIERDSDDRKTFRFTRDELGLLVDFLGEVFKREGNQVIGLRGMPRVGKTESIIAGSVCSNKRWTFVSSTL
LRQTVRSQLSEDEMSMNNVFIIDGIVSTIRSNEKHFSLLQEIMATPSTKVIEHPDIFIKESHYTYNDFDYIIELRNNPDE
EINYDSFTYNTEPF
>lcl|NZ_CP026520.1_prot_WP_042233106.1_4164 [locus_tag=PC41400_RS21015] [protein=hypothetical protein] [protein_id=WP_042233106.1] [location=complement(4674828..4675694)] [gbkey=CDS]
MSFSRVKLALFLPVITVMLSSCGTRGLPTITEQQLASKHSVLVVTASDVTQETKTSVAAALAKWRSGSGLTSEWKQDAAF
DSAMGTQLESKRYDYVVVIGSSLVPQASAEAVRLKNQKFVVLADRLEDPGMQAEKTAPPNTVLRGVDHAMIEQAMSGWVK
DQADKRKSIEWVTKSSVPIPSGLSPSGLSGHTVNTDNNPEWFDQLKKQAAEHRSSWIIQYTALQKTDQQKIKSLGIPVVP
LDQSFAVQVNWEAVMAQEQNRMIQGDWKPGIQTYSAEEMKQVEFLKKS
>lcl|NZ_CP026520.1_prot_WP_042233104.1_4165 [locus_tag=PC41400_RS21020] [protein=DUF3243 domain-containing protein] [protein_id=WP_042233104.1] [location=complement(4675775..4676026)] [gbkey=CDS]
MSTVLTVFDRWKEFLAERVDQAERVGLNDEKISNLAFQIGEFLSDKVDPENREERLLKELWDAGDENERKVMARLMVKLV
NKE
>lcl|NZ_CP026520.1_prot_WP_042233101.1_4166 [locus_tag=PC41400_RS21025] [protein=SDR family oxidoreductase] [protein_id=WP_042233101.1] [location=complement(4676152..4676907)] [gbkey=CDS]
MTGLRPFTEQTVLVTGGSRGIGAAVAERFAKLGMNVVIHYLQSHEAANEVARRCMNHGAKVMTVCADLNSKEQIVAMRDK
LEQRGFIPDVLVNNAGISHYGLLSDVTEEEWDRVMDINLKGTFLCSQLFMDRMISRKYGRIINVSSIWGISGASCEVIYS
TAKGGINAFTKALAKELAPSGVTVNAVAPGAVDTDMMSGFDPAEKEAIASEIPAGRFAAPDEIASLVYFLALPESGYITG
QIISPNGGWLT
>lcl|NZ_CP026520.1_prot_WP_042233099.1_4167 [locus_tag=PC41400_RS21030] [protein=pitrilysin family protein] [protein_id=WP_042233099.1] [location=complement(4676904..4678190)] [gbkey=CDS]
MNVIPYPHLNETLYEEKLPNGLHVYVLPKEGFQKTYATFSTRYGSIDNHFQVEGGPEIRVPDGIAHFLEHKMFEEPTGDV
FATFASRGASANAFTSFERTVYLFSATGQIMENITTLIDFVQNPYFTEENVEKEKGIIGQEINMYRDNPDWRAYYGLIEA
LYSKHPVRIDIAGTIPSITEITKDMLYDCYHTFYHPSNMTVFIAGGVNPEEVISHIRANQAAKTFKPQGEIRRFFESEPE
SVETEHKETLLPVSLPKCLFGFKEKALHEEGDAILKRELTTKLVLEVLLGASSPIYQKLYNEGLISDNFSHEYNSSPEYA
FSAMGGDTKDPDLLLSRIREELDRVKAEGLDEEAFERSRRKKIGGFLRMLNSPEAITNEFTRYLFKGADMFDILKTYESI
TKDEAESRLREHFDWSRFAVSLVRSERS
>lcl|NZ_CP026520.1_prot_WP_042233098.1_4168 [locus_tag=PC41400_RS21035] [protein=pitrilysin family protein] [protein_id=WP_042233098.1] [location=complement(4678187..4679473)] [gbkey=CDS]
MSTQQFERSTLRQIRLHVLPTDRYKTFAISAYIGRPLSEEDVTQTALVPFVLRRGTESYPETKQFREHLDDLYGAGFGFD
IYKRGDYQIVQFRMDVINDRFVSSPESLLEKALSFLGEAITKPAAEGDAFLSKYVESEKETLRKRIESIINDKIRFAAER
CIAEMCKGEPYRLHPLGELKDIPASTPESLFAYFKKWLAEAPIDLYVVGNTTLQEVENIVSRSFRLEREENPGYTRTAPQ
RRTGETHVVVERFPVAQGKLNMGLRTPITYGDDAYPAALMYNGILGGYPHSKLFLNVRERASLAYYASSRLDGHKGILTI
QSGIEMQNYEKAVDIIRKQLEDMRTGFISDIELSQTRAMIANQLREMQDSAFEQIAFDFNAVLSGKERTVQELIDGVTNT
SIAEIQQAAEGVFLDTIYFLRDEKEEAK
>lcl|NZ_CP026520.1_prot_WP_042233294.1_4169 [gene=sleB] [locus_tag=PC41400_RS21040] [protein=spore cortex-lytic enzyme] [protein_id=WP_042233294.1] [location=complement(4679609..4680397)] [gbkey=CDS]
MNKRLIAITVCVVFTLFAALQIKQNFTTEATFSKNLISYGAKGSDVNELQGRLKFLGFYTGAVDGDYGWRTLRSVKWFQG
EFGLKVDGHVGPKTKLKLWEATKDWKPTAAEAGNTGKAAGPANQANQGGAAASGSNLTPTTHKGFSKSDLDLMANAVYGE
SRGEPYIGQVAVAAVILNRVDSASFPNTVSGVIFQPGAFTAVADGQIWLTPNETAKKAVRDAINGMDPTGGCEYYFNPET
ATSKWIWTRPQVKTIGKHIFCK
>lcl|NZ_CP026520.1_prot_WP_042233096.1_4170 [locus_tag=PC41400_RS21045] [protein=DNA translocase FtsK] [protein_id=WP_042233096.1] [location=complement(4680533..4683130)] [gbkey=CDS]
MAKRRKKSKTVKTNLMFELYGILILIFSVIALAREGHVGRSFHYVCRFFIGTWDFLIPLICIYIALHVMVKRSWPSRWTT
RKTGVLLIVAGLLVMNHINLFQALFPKGNYEATTILSVTWDALKQGLQPANGKILLNDVGGGMIGAVIYSLLYFLFDTLG
ARLVEFALFAIGLILVTNLSYVEAMQKLRARGKRFFTLLEKKFQTMQRGKGKSSSASKTKGKAAVSATKSERTYKEPDFP
YDDEFDEEVYRPDPKRKKPRFLQLLKQPDPQQDDDQDEDDLAVSGTAGGYSEEKVVYKVDSGRDSAALDMNDEPQADREF
STVFRDFQEQVQKEEKVVPSSPAAEGTVGPAVVAGSPKAAASLHAEKIPADEQLETLDDLQLVDTAVPPVLPPYDLPSLT
LLSKPAAGKGSEMADYKANARKLEATLESFGVRAKVLEVVRGPAVTRYEIQPDVGVKVSRIVSLTDDIALALAAKDIRME
APIPGKSAIGIEVPNSEVSVVTMREVMETSAFQEASSRLSVTLGRDISGQPIVGNLAKMPHLLVAGATGSGKSVCINGII
TSILYKAKPNEVKFLMVDPKMVELNVYNGIPHLMAPVVTDPRRASLALKKVVVEMEKRYELFSKSGTRNIEGYNAMLIEN
GTEAPLPYIVVIVDELADLMMVAANDVEDAICRLAQMARAAGIHLIIATQRPSVDVITGVIKANIPSRIAFGVSSQVDSR
TILDMVGAEKLLGRGDMLYLPVGASKPVRVQGAFLSDQEVETVVNFVRTQQEANYREDMVPQVEEQQEQQDEFGDELYDQ
AVQIVIEAKQASASLLQRRMRIGYARAGRLIDSMESNGIVGPHEGSKPREVLISLEQFQHNRMSS
>lcl|NZ_CP026520.1_prot_WP_042233094.1_4171 [locus_tag=PC41400_RS21050] [protein=YlzJ-like family protein] [protein_id=WP_042233094.1] [location=complement(4683195..4683425)] [gbkey=CDS]
MTHYTILPDEVIYEGLDEHEPRFRDVEISGILMQVEAVDESRVRIVRLYSPEPKHYLNPLFTPGSLINYVPGPQGI
>lcl|NZ_CP026520.1_prot_PC41400_RS30110_4172 [locus_tag=PC41400_RS30110] [protein=ClpP family protease] [pseudo=true] [partial=5'] [location=complement(4683422..>4684111)] [gbkey=CDS]
EEQKKLAAVETIQQLGQINTPAGESNIYCMSIIGQVEGHIVLPPQNKTTKYEHLIPQLVAAEQNPKIEGILVILNTVGGD
VEAGLAIAEMLATISKPTVTLVLGGGHSIGVPIAVSSNRSFIAETATMTIHPIRLTGLVIGVLQTFEYLEKMQERVTRFV
TKHSKISEEQFKELMFKTGELTRDIGTTVIGQDAVKYGLIDEVGGIGQAMAELNRRIEERRGSQGGGTA
>lcl|NZ_CP026520.1_prot_WP_042233090.1_4173 [locus_tag=PC41400_RS21060] [protein=ribonuclease J] [protein_id=WP_042233090.1] [location=complement(4684803..4686479)] [gbkey=CDS]
MSKKNNDKVLIFALGGVGEIGKNMYVVQYANDIVVIDSGLKFPEEDMLGIDIVIPDINYLLENKDKVRGILLTHGHEDHI
GGLPYVLRHLNVPVYGTRLTLGLVENKLREANLLGETKRILINADSELNLGTITATFFKTNHSIPDSVGVALETPEGVIV
HTGDFKFDQTPVNDQYADLHRMAEIGKKGVLALLSDSTNAEKPGFTGSEKSVGVTIEEVFRKAKQRVVIATFASNIHRVQ
QVIDAAEATRRKVTVIGRSMVNVVSIASELGYLRVPDGMLIEPEEVNRMAADRVVVLCTGSQGEPMSALTRMARSTHRKV
DILPGDTVIIAATPIPGNERYVGRTVDELFRLGADVIYGPGSKTGVHVSGHGSQEELKLMLNLMKPKYFIPIHGEYRMLR
LHGQLAESVGVDQEDIFILDNGDTVEIQGGVARKGARIQAGNILIDGLGVGDVGNIVLRDRKLLSQDGILVVVVTLSKQD
GTILSGPDIISRGFVYVRESEGLLDEANRIVTSTLNKLMNDNVNEWASLKTHVKDALGRFLYEQTRRRPMILPIIMEV
>lcl|NZ_CP026520.1_prot_WP_042233088.1_4174 [gene=dapA] [locus_tag=PC41400_RS21065] [protein=4-hydroxy-tetrahydrodipicolinate synthase] [protein_id=WP_042233088.1] [location=complement(4686669..4687529)] [gbkey=CDS]
MDFGRVITAMVTPLNEQLEIDWDQVGPLVDYLINEQGSEGLVVCGTTGESPTLTDEEKLKMIEAVVKHANGRCKVIAGTG
SNNTHHTIELSQEAAKLGVDAILLVAPYYNKPSQEGLYQHFKAVAEAVDVPIILYNVPGRSIVNIDADTTIRLSHIPNIV
ATKDCSGTDQLTIVAAGAAEGFTVYSGDDSMTLPALAVGCHGIISVASHVVGKEMQELVEAYTSGDVKRAARLNAKLLPV
FTGLFCAPSPAPVKYALNLKGVNTGSVRLPLIDLSEQEQAFVASLF
>lcl|NZ_CP026520.1_prot_WP_042233085.1_4175 [gene=dapG] [locus_tag=PC41400_RS21070] [protein=aspartate kinase] [protein_id=WP_042233085.1] [location=complement(4687559..4688776)] [gbkey=CDS]
MRIRVQKFGGTSLSTPEARKHVLEHIQAARSEDCQLVVVVSAMGRQGEPYATDTLLGLIRQNGDQLPARERDLLMCCGEW
ISAANLCNLIQAEGIPAVVLTGGQAGIETNDAYGNAQIVSVQPKRILEHLDRGEVVIVAGFQGQTADHEFTTLGRGGSDT
SATAIGGALKAEVVDIFTDVNGILTADPRIVEDARPLTRVSYAEICNMAHNGAKVIHPRAVEVAMQANVPVRVRSTFSKE
EGTLVTNPDSLYGAEKRVEDRHVTGIAHFGNITQIQVPAAEGQYDLQLQVFKTMAQHSISVDFINVNPSGVVYTVFDHVA
EKALDLLRGLGYKPTSLSGCAKVSVIGGGMNGVPGVMAKIVEALTEENIRILQSADSNTTIWCLVHGEDMVSAVRALHKS
FQLHK
>lcl|NZ_CP026520.1_prot_WP_042233083.1_4176 [locus_tag=PC41400_RS21075] [protein=aspartate-semialdehyde dehydrogenase] [protein_id=WP_042233083.1] [location=complement(4688872..4689909)] [gbkey=CDS]
MSNEKTFNVAVVGATGAVGEQIIRLLDERNFPIKELKFLSSPRSAGTKIRFKDRDITVEAATPDSFKGVDIALFSAGGDV
SKELAPIAAAGGTICIDNTNAFRMDPQTPLVVPEVNIEEVHKHKGIIANPNCSTIQMVAALKPLYDRFGISKIIVSTYQA
VSGAGSKAIDEMLRQSREALAGNPVNPDILPVGSLPVKHQIAFNAIPQIDKFLDNGFTNEEMKMINETKKIMNDESLEVT
ATCVRIPVVYGHSESVYVELKEDFELEEVRSLLADAPGIVLVDNPQEQQYPLATEATGKLETFVGRVRRDLHNKRALNLW
IVSDNLLKGAAWNAVQIAEYFAQNK
>lcl|NZ_CP026520.1_prot_WP_042233081.1_4177 [locus_tag=PC41400_RS21080] [protein=dipicolinate synthase subunit B] [protein_id=WP_042233081.1] [location=complement(4689945..4690541)] [gbkey=CDS]
MNWNGVTVGFALTGSHCTLEEIMPQIQRFIDAGATVVPIVSQSVLTTDTRFGKAVDWQQQLRDMTGSDPITTIVEAEPLG
PSKKLDVMVIAPCTGNTTSKLANAMTESPVLMAAKAQMRNQRPLVLAISTNDGLGLNSANIAKLLVAKNIYFVPFGQDAP
EAKPNSLVARMDLISETCEAALNGKQLQPMIIERFQYS
>lcl|NZ_CP026520.1_prot_WP_042233080.1_4178 [gene=dpsA] [locus_tag=PC41400_RS21085] [protein=dipicolinate synthase subunit DpsA] [protein_id=WP_042233080.1] [location=complement(4690538..4691437)] [gbkey=CDS]
MLTGAEVIFIGGDARQLEVIDKLIELDASVTLIGFDNLQKPFAGAAGKRLSPEALEKADALVLPVVGTDEQGRVESIFSG
NELTLTADLLGALPKHAVIYTGMAKPYLKQLCESQGLKLVELLERDDVAIYNSIPTAEGALMIAIQNTDITIHGSQSMVL
GFGRTGLTMARTLQGLGAKVKVGVRKPEHFARALEMGFTPFYTRELYQEVANIDLLFNTIPTMIVTAQIITNIPHRAVLI
DLASKPGGMDFRFAEKRGIKAMLAPGLPGIVAPKTAGRILANTLSQLMVEDIRNGSGPE
>lcl|NZ_CP026520.1_prot_WP_231392056.1_4179 [gene=dut] [locus_tag=PC41400_RS21090] [protein=dUTP diphosphatase] [protein_id=WP_231392056.1] [location=complement(4691748..4692149)] [gbkey=CDS]
MSAAASGFDLHAANIEPLDLAPGERALVPTGFALAMPVELEAQIRPRSGLAYKHGITTLNSPGTIDADYRGEVKVLLINH
GQETFTIRRGERIAQMVFQLVPAVLLTEVDELDDTTRGAGGFGHTGLTTEARK
>lcl|NZ_CP026520.1_prot_WP_042233291.1_4180 [locus_tag=PC41400_RS21095] [protein=pitrilysin family protein] [protein_id=WP_042233291.1] [location=complement(4692217..4693473)] [gbkey=CDS]
MNKTHLNNGLRVVTEKIPTCRSVAIGIWVKTGSRNETQLNNGISHFIEHMLFKGTERHSAKDIAEVFDGIGGNVNAFTSK
EYTCYYAKVLDEHLGIAVDILSDMFFNSVFNEEELEKEKNVIFEEIAMYEDTPDDLVHDLVAKAAYGSHPLGYTILGTQA
HLSGMKPNDLRAYMKECYTIENTVIAVAGNIDDSVFELLEKHFGHFNVSGTQQNQEEPAFMSERVFQAKTTEQNHICLSL
PGLSLREENLYPMVLLNNAIGGGMSSRLFQEIREKRGLAYSVYSYHSSHIDNGVFTIYTGTAPKQTEEVLKVTIEMLNDI
AEKGMTDAELRKGKEQLKGSLILSLESTNSRMNRIGKNELMTGRHYTLDEMINRIESVEMEHIHALTRKLFSKPFAIAMV
GSRDEALQSIRGDELLGK
>lcl|NZ_CP026520.1_prot_WP_042233076.1_4181 [locus_tag=PC41400_RS21100] [protein=polysaccharide deacetylase family protein] [protein_id=WP_042233076.1] [location=complement(4693564..4694625)] [gbkey=CDS]
MQKKKLVLLGGITAGCIWVLLQSETVHTFVGAIQQGKQPSVETQTKLSNVAGGPESNEKTAQPAEDALLGRIREEAKGLY
IAPIDAKVDRIWKAVPGYNGLEVDIDKTYEAAKKNGAGDKLPFVMKEVPPKVHLNDLPPEPIYKGNSKKPMVSFMINVAW
GEEFLPSILETLRKENVRATFFFDGSWLKKHVDIAKEIQKQGHELSNHAYSHKNMSTLSRSKAYEEIVKTEDLLKKELGV
TNVLFAPPSGDFDAETVKIAAELKLKTVMWTLDTVDWKNPGAPAILAKVRKNIEPGSLILMHPTQSSSSALPEMISIIKG
KGLHLGTVSETLSPQRISAPAPGAGQVGDERSG
>lcl|NZ_CP026520.1_prot_WP_042233073.1_4182 [gene=pnp] [locus_tag=PC41400_RS21105] [protein=polyribonucleotide nucleotidyltransferase] [protein_id=WP_042233073.1] [location=complement(4694785..4696890)] [gbkey=CDS]
MEKRVQMDLAGRPLILETGRLAKQANGAVLARYGDTVVLCTVTASSEPKDLDFFPLTVNYEERLYAAGKIPGGFIKREGR
PSEKAILSSRLTDRPIRPLFPEGFRNDVQIVAVVMSVDQECPPEITAMIGVSAALSISDVPFSGPVGGVIVGRIDGNFVV
NPTPAQEDKSDVHLVVAGTKDAIMMVEAGAAEVPEDVMLEAIMFGHDEIKKVIALQEQLVAEAGVPKMEVKLHSVDGEIS
TAVREFAEARLVEAIKVQEKHARQEAIDAVNADAVAHFTALYEETPEKLKDVKEVLHDIVKDEVRRLITHDKVRPDGRAL
DEIRPIDVDINLLPRTHGSGLFTRGQTQALSVCTLGALGDVQILDGLDLEETKRFMHHYNFPPFSVGEARPLRPPGRREI
GHGALGERALSYVIPSEAEFPYTIRLVSEVLESNGSSSQASICASTLAMMDAGVPITAPVAGIAMGLIKEGEHFSILTDI
QGMEDHLGDMDFKVAGTSKGVTALQMDIKIDGINRDILKQALDQAKAGRTFIMNKMLERISEPKKELSAYAPKIVVMQIN
PDKIRDVIGAGGKIINKIIEETGVKIDIEQDGKVFIASSNAEMNERAKQIIEGIVRDVVVGETYLGTVKRVEKFGAFVEV
LPGKEGLVHISQLSTERVAKVEDVVKVGDTITVKVTEIDNQGRVNLSRKATLVNNEVSTQS
>lcl|NZ_CP026520.1_prot_WP_009674959.1_4183 [gene=rpsO] [locus_tag=PC41400_RS21110] [protein=30S ribosomal protein S15] [protein_id=WP_009674959.1] [location=complement(4697142..4697411)] [gbkey=CDS]
MALTQERKQELIQTHKTHESDTGSPEVQIAILTENIVNLTSHFRTHKKDHHSRRGLLKMVGQRRKLLAYLKNKDVKRYSA
LIEKLGLRR
>lcl|NZ_CP026520.1_prot_WP_042233071.1_4184 [locus_tag=PC41400_RS21115] [protein=bifunctional riboflavin kinase/FAD synthetase] [protein_id=WP_042233071.1] [location=complement(4697565..4698509)] [gbkey=CDS]
MQTIHLSYPIENAELLPLAGKKVIAIGDFDGVHLGHREVIQRAVENARELGIPAAIMTFDPHPRELLGQEKYARLLTPTS
YKMELFEKLGVDCTYVLKFDTALMQASPQQFVDRVLEALGVETVVVGFDFTFGYKGRGTPDTLCELGHGRFAVEVIRPYR
MNGAKVSSTAIREALLTGHPPGASELLGRGYGIRGVVVSGDKRGRQLGFPTANLRMSENFLVPASGVYAVRAFVGGSWYR
AVMNIGTKPTFHSGEVMSLEVHLLDFDQDIYGSTVFVEFVDYLRPERKFNGIDELITQLKRDVATAEAKFAQIL
>lcl|NZ_CP026520.1_prot_WP_042233069.1_4185 [gene=truB] [locus_tag=PC41400_RS21120] [protein=tRNA pseudouridine(55) synthase TruB] [protein_id=WP_042233069.1] [location=complement(4698596..4699531)] [gbkey=CDS]
MTIEGILPILKPKGMTSHDVVAKARRFLGIKRIGHTGTLDPEVTGVLPLCIGRATRMVEYIQELPKQYEAVMVFGLATDT
EDLSGNVLEEIPHVQLAEDKVRSVIQSFVGEISQVPPMYSAVKVDGKRLYELARAGVEVERKARNVTIYEIELLELDLNR
YHPEVRFRVLCSKGTYIRTLCVDIGRALGVPAAMKELVRSSTGRIGLEQCVALEDLPRLREEGLLDTKIIQADQAVDYLP
SLVLNEQLSVFALQGRKVRLHEKPALHGSHDSETGVEQTGIIRVYDPHKRFIGLFSWNETEGHLAPVKVFN
>lcl|NZ_CP026520.1_prot_WP_042233067.1_4186 [locus_tag=PC41400_RS21125] [protein=bifunctional oligoribonuclease/PAP phosphatase NrnA] [protein_id=WP_042233067.1] [location=complement(4699528..4700541)] [gbkey=CDS]
MTAGQFIGGGASGSYLEQLEAAKQFMHDHNDFLVVSHVQPDGDAISSTLAIAWLLDRMGKRYTLANEGDIPEKFSRLWGF
DGIRTLSQLSDHLSYDTVISVDCADFSRIGRVSEKFTQGAKLLNIDHHATNDRFGECSVIQASAAATVEIIYDLSRHLQM
EADVDFGTCIYTGLLTDTGGFRYSSTTPKVMKTAADLLEWGVKGSELAEAFLERMTFQQIRILQQGLATLDFHNNGRIST
IVVSTEVIEATGASNEDMDGLVNYARNVEGVEVGLLFKQRNESAVKVSLRSAGAVDVAQIAQSFGGGGHVRAAGCTINGS
IEEALRRVVEEVGNKLS
>lcl|NZ_CP026520.1_prot_WP_009672009.1_4187 [gene=rbfA] [locus_tag=PC41400_RS21130] [protein=30S ribosome-binding factor RbfA] [protein_id=WP_009672009.1] [location=complement(4700538..4700894)] [gbkey=CDS]
MARVRVGRVGEQVKKEISQILQSEIKDPRIGFLTVTGVDVTNDLSQAKVYLSVLGTDEQKEGTLKALSSASGYIRSELGK
RIRLRKVPELIFKFDTSIDYGSRIESLLHDLNDGKSAE
>lcl|NZ_CP026520.1_prot_WP_042233063.1_4188 [gene=infB] [locus_tag=PC41400_RS21135] [protein=translation initiation factor IF-2] [protein_id=WP_042233063.1] [location=complement(4700901..4703642)] [gbkey=CDS]
MGLTNKQDPKDKLRVYEYAKSLNMSSKEIITILKRLNIPVNNHMSVMENNAVSSVENFFRDIKANAAAKRAGSDGGNVSP
NSNQPKPDKQSNPKAEVSAKQPVQQTGSEQAKKEERPAAEARPQGSQDNRAPRPSGDGRTGAPRGDRPQGQGAPRSGGQG
GGQRPQGGQGGYNRGPQSGGQGRPQGGQRPQGQGGAPRSGGQGGGPRPQGGQGGGFNRGPQGGSGGQRPQGGGQGGQNRG
GSAGAGARPGQAPRRTDGRPAGGADAGRGGQRTGDNRQGNRDDRSKSSRFDDGRGGRGGAGNRGGNNRGGRNNGRGRGQE
PMVKKEKIDNTPKKIIVRGAMTVGELAKLLHKDVSEVIKKLLFLGVMATINQELDLDTIQLVAGEYNVEVDLKIRVEDDR
FELEEEVDDPADLKERPPVVTIMGHVDHGKTTLLDAIRHTSVTEGEAGGITQHIGAYQVEANGKKITFLDTPGHEAFTTM
RARGAQVTDITIIVVAADDGVMPQTVEAISHAKAAGVPIIVAVNKIDKPEANVEKIKQALTEYELVPEEWGGDTIFCEVS
AKQRIGLENLLEMILLVAEVQEYKANPDKRARGTVIEAELDKGRGPVARILVQHGTLKVGDSFVAGVCFGRVRAMVNDKG
KRLKEAGPSTPVEITGLTEVPQAGDPLLAFEDERKARDIADKRAITLRQSELTANSRVTLDDLYKHIKDGEIKDLNVIIK
ADVQGSLEALRGSLEKIEVEGVRVKMIHTGVGAITESDIILASASNAIIIGFNVRPEPAAQVTADQEKVDIRLHRIIYNV
IEEIEQAMKGMLDPIFKENIIGHAEVRNTFKITGVGTIAGCMVTSGKIARNAEVRLLRSNIVVYEGKVDALKRFKDDAKE
VAQGYECGISLEKYNDIKEGDIIEAFVMETVER
>lcl|NZ_CP026520.1_prot_WP_042233061.1_4189 [locus_tag=PC41400_RS21140] [protein=YlxQ family RNA-binding protein] [protein_id=WP_042233061.1] [location=complement(4703656..4703958)] [gbkey=CDS]
MTNKNYFSNLGLAMRAGKLVTGEESVLKAIRSGEAKLVILAEDAADNTSKKFQDKCRSYDVPLLVLGNREELGSSIGKEA
RVTIAVVDDGFARMLKKYRV
>lcl|NZ_CP026520.1_prot_WP_042233058.1_4190 [locus_tag=PC41400_RS21145] [protein=YlxR family protein] [protein_id=WP_042233058.1] [location=complement(4703948..4704259)] [gbkey=CDS]
MKQRKIPLRKCVACQEMMPKKELIRVVRTPEGEIQIDLKGKKSGRGAYLCGKVACFRLAKKSKALDRALKQPVGAPIYDQ
LEQDFIQVEEEFHAGKDGMDDDE
>lcl|NZ_CP026520.1_prot_WP_009672008.1_4191 [gene=nusA] [locus_tag=PC41400_RS21150] [protein=transcription termination factor NusA] [protein_id=WP_009672008.1] [location=complement(4704286..4705383)] [gbkey=CDS]
MNMDFIEALAEIEREKGISKDLLIDAIEAAMISSYKRNFNTAQNVRVDINRQTGTIRVYARKTVVEDVLDPRLEITLHAA
REINQNYTMDDIVEIEVTPRDFGRIAAQTAKQVVTQRIREAERGLIYNAFIEKEEDIVTGIVQRQDQRHYYIDLGKVEAV
LPLTEVMPNEKFKQGERVKAYITKVENTTKGPQIILSRTHPGLLKRLFELEVPEIYEGVVEIKSVAREAGFRSKIAVHSR
NTEVDPVGSCVGPKGLRVQTIVGELRGEKIDIVRWMESVEEYVANALSPSKVLEVNVFENEKMARVIVPDYQLSLAIGIK
GQNARLAAKLTGWKIDIKSEAQAEQEYGRPRSQIEEMHQDSVSFD
>lcl|NZ_CP026520.1_prot_WP_042233055.1_4192 [gene=rimP] [locus_tag=PC41400_RS21155] [protein=ribosome maturation factor RimP] [protein_id=WP_042233055.1] [location=complement(4705432..4705890)] [gbkey=CDS]
MNSPIKSVVEDMLLDYLNEQGFELVDIEYVKEGSNWFLRVFVDKEGGIDIDDCGRISEFLSSKLDENDPIPTAYFLEVSS
PGAERPLKKPEDIKKAVGKHVFVTTYEPIDNSKEFEGELLSFDGEELVVQTPKKRSAIPYSKVASARLAIVF
>lcl|NZ_CP026520.1_prot_WP_042233053.1_4193 [locus_tag=PC41400_RS21160] [protein=PolC-type DNA polymerase III] [protein_id=WP_042233053.1] [location=complement(4706094..4710398)] [gbkey=CDS]
MSHTPDKRQRFELLMKQADISHEMSTVHFADGYIDKVEISRKNREWTFHLVKQELVPQDVYRSFCRNVQEKFAHIAKTRF
VIHMENAEPGRLVEEYWSLFLEWVQREAVTVNGWLSKAKTEVQGNQLTLILLDQIGFELAKKKNMDNWVKSFFQDYFKTE
FTVKYALNEKSTEEFERFAQQIVEEEKLVTQEIMQSIEQELQASAIPDADVRLMVGYDIKDAPVPIQDIVEEEKKVTIQG
AVFNLEVKELRNGSTLYSFNVTDYTDSIMLKSFGKTKEDVRILNLLSNGCWIRARGRVEYDRFMMVPELVMIPSDINEVL
PPKGRMDDAEEKRVEFHLHTTMSTMDGITPVDQYIKTAAKWGHKAIAITDHSGVQSYPDANKAGKKNGIKVIYGVEANVV
NDSVPIVLNPIDADLKQATYVVFDIETTGLSVLNNKIIEIAGVKVQEGKEIDRYSTFIQPHEKIPYHIQQLTNITDEMVK
DAPELADELPKFIDFIGDSVLVAHNARFDIGFIQANLKRLGHPQMTNAVLDTLEMARLLFPSMKNHRLNTLSDKFKVNLE
NHHRAIDDTIALGHVLQHLINEAYDRNITNLGRMNDFVGKDLSNQRPFHCCIYAANSVGKKNLFKLISISHTEYFNKVAT
IPKSRLIELREGLLIASGCEKGEFFETVLNKSQEEAEHVAEFYDVLEIQPVSYNMHLVDKGLVGSRADLEDAVKRIVAIG
AKQGKPVIATGNVHYLEPRDKVARDITINGITGFSPLKDMKKPDAHFRTTREMLEEFRFLGQEKAHEVVVRNTNDLADRF
EEIELFPTKLFTPIIEGADDEIRNTCYDTARQIYGDEIPEIVVQRLEKELVPIIKYGFSANYLISERLVKKSNEDGYLVG
SRGSVGSSVVATFLGISEVNPLPPHYICRSCKFSEWFTDGSIPSGFDLPDKTCPTCGDKLKGEGQDIPFETFLGFKGDKV
PDIDLNFSGEYQPVAHNYTKVLFGEKSVFRAGTIGTIADKTAFGYVKKYEEGQGKKWRNAEIVRLASGCTGVKRSTGQHP
GGIVVVPDYMEVDDITPVQFPADDKNSEWKTTHFDYHAFDANLLKLDILGHDDPTMMRMLQDLTGIDPTSIPMNDPKVMS
IFHSTDALGVPPEKIRTPVATYGVPEMGTKFVRQMLQETHPSSFADLLQISGLSHGTGVWLGNAQELIKKGICNIKTVIG
CRDDIMLYLIYKAGMDAGLAFKITESVRKGKGLTDEWKEDMKKHNVPAWYIESCERIEYMFPKAHAAAYVISAVRTAYFK
VYHPIAYYATYFTVRAEDFDLELLCQGYDAIMKKLIEIEEKGFQALPKEKAMISILEMALEMTARGFSFKPIDLYRSDAT
KFIIDGDSLIPPFSAVPGIGENAARNIAASKDSGDFLSIEDFQNRSKASKTVVELLNGMGCFRGLPESNQLSLF
>lcl|NZ_CP026520.1_prot_WP_042233052.1_4194 [gene=proS] [locus_tag=PC41400_RS21165] [protein=proline--tRNA ligase] [protein_id=WP_042233052.1] [location=complement(4710516..4711964)] [gbkey=CDS]
MSKDKQFVREITPQGEDFSQWYIDVIKKADLMSYSPVRGCMVFKPDGYEIWENIQRELDKRFKETGHRNAYFPLFIPESF
FQKEKEHVEGFNPELPWVTEAGGEKLEERLAIRPTSETMIGHMYAEWINSYRDLPLLINQWANVVRWEKRTLPFLRTTEF
LWQEGHTAHEDEEDARRETMQMLEIYRDFAENFLAMPVIIGQKTPSEKFAGAVDTFSIEAMMKDGKAVQAGTSHYLGTNF
AVAFDIKYLNRENQHQFVHTTSWGVSTRLIGATIMAHGDDRGLALPPKVAPTQVIMIPIGPPKTREQVVARTDELFAELK
QAGVRVRVDDRADLSPGWKFNEYEMRGVPIRVELGPRDMENGQIVLVSRITGEKKIVQQVNFVQEVQQLLEEIHQEMFNR
AKQHLEEHLITVDTLDEFKQYLETNRGFALAGWCGSEACEAQVKEEAGATSRNIPFSPRESKETCLVCGDKAKHTVVFGK
SY
>lcl|NZ_CP026520.1_prot_WP_042233050.1_4195 [gene=rseP] [locus_tag=PC41400_RS21170] [protein=RIP metalloprotease RseP] [protein_id=WP_042233050.1] [location=complement(4712074..4713324)] [gbkey=CDS]
MGSIETGLKIVLMFFVLVTIHEWGHFYFAKRAGILVREFAIGFGPKLFSYKKGETRYTLRLLPFGGYVRMAGEDPEIVQI
QDGQTVAVKLNKDNKVTKLYLDQLDQRANVIQGSVQQIDLEKDLFVKLEVGDEQVRYAVQPDALMVTRGVETQIAPLDRQ
FGSKTVGQRFLSILMGPVMNFLLAAVLFLTLVIMQGVPTNVKVDKLEPGRPSEKAGLMAGDVIVSVNQETIGDNRENFMK
VVGSSPNQKMNWVIQRGGAEKKIEITPDENGKVGVYITGDKRSASFGETLKGTCNYVSTSTVGILDGFKKLVLGQFKMDD
LGGPVRIVQVTGEFASHGSESYIYWVALLSLYLGLFNLLPFPALDGSRLVFLAIEAVRGKPVDPNRESMVHFIGFAMLML
LMVAVTYNDILRLIKS
>lcl|NZ_CP026520.1_prot_WP_042233047.1_4196 [locus_tag=PC41400_RS21175] [protein=1-deoxy-D-xylulose-5-phosphate reductoisomerase] [protein_id=WP_042233047.1] [location=complement(4713524..4714669)] [gbkey=CDS]
MNKRIAILGSTGSIGTQTLDVIAHHPDQFTVEALAGGRNVELLAEQIRRFKPKLASVATRELADQLSALIPAGTRILFGE
EGLLETAAGTDADFVVTAVVGNQGLEPTIAAIEAGKTIGLANKEPLVSAGHLVTEAANRKGVRLLPIDSEHSAIFQCLNG
ERREEVRKITITASGGSFRDRTRAELEGVTVEQALNHPNWSMGAKITIDSATMVNKGLEVIEAHWLFDLPYDQIGVILHP
ESIIHSYVEFKDSSIIAQLGNPDMRVPIQYALSYPNRLLSSAKSLDLAEIATLNFRPMDFDRFPCLRMAYESGRTGGTAP
TVFNAANEQAVAKFLQGEIPFLRIETIIEECLQRHSVMSSPQIGDILEADRWARETASLLG
>lcl|NZ_CP026520.1_prot_WP_009676789.1_4197 [locus_tag=PC41400_RS21180] [protein=phosphatidate cytidylyltransferase] [protein_id=WP_009676789.1] [location=complement(4714727..4715524)] [gbkey=CDS]
MKQRILTGAIAGAGFLGLLAVGGNWFAALIVLMAIIGFDEYVRMNKLKDHRITWLVGMIALILLTLPWEMHRFTYELNEV
GLIWGAAFVLMAATVLSKNKITIDMVSLLFLGVVYIGVGFHYMIMTRWIPEHGLFWTLLIFVCIWATDSGAYFVGSKFGK
HLLWPVISPKKSVEGALGGVLIAIVAGLCFSLARPELLSPARAVLIGVVIAVSGQLGDLMQSAYKRVKGIKDTGTILPGH
GGILDRVDSWLIVFPLVHWLALIPS
>lcl|NZ_CP026520.1_prot_WP_042233045.1_4198 [locus_tag=PC41400_RS21185] [protein=isoprenyl transferase] [protein_id=WP_042233045.1] [location=complement(4715549..4716319)] [gbkey=CDS]
MFNQLKKWLGNESAQDEKSPHLELDNVPSHIAIIMDGNGRWAKQRGLPRVAGHHSGMKNVKKITIAADSIGVKVLTMYAF
STENWKRPKEEVEFLMKLPQEFFPLEIEELIERNVRIRMTGWKESLPDYTLNAIEDAIERTKNNTGLILNFALNYGGRKE
LLAGVQDVLKDVQEGKLRPEELTEETFSSYMLTSGLPDPDLLIRTSGELRLSNFLLWQLAYTELWFTEAYWPEFSEEHFY
RAVAEYQRRARRYGGL
>lcl|NZ_CP026520.1_prot_WP_042233043.1_4199 [gene=frr] [locus_tag=PC41400_RS21190] [protein=ribosome recycling factor] [protein_id=WP_042233043.1] [location=complement(4716383..4716937)] [gbkey=CDS]
MPQSIKKNAEERMSKAIDALKRDLATLRAGRANPALLDKIQVEYYGAMTPVNQLANISTPDSRTLIIQPWDKSSMGAIEK
AILKSELGLTPANDGTVIRLVIPPLTEERRSDLVKLTRKYGEEAKVAIRNIRRDANDDIKKQEKNGISEDESRRHQDDIQ
KFTDKFTAEVEKVLTGKEKEIMEV
>lcl|NZ_CP026520.1_prot_WP_042233041.1_4200 [gene=pyrH] [locus_tag=PC41400_RS21195] [protein=UMP kinase] [protein_id=WP_042233041.1] [location=complement(4716942..4717667)] [gbkey=CDS]
MERPLYKRVILKLSGEALAGQQGFGIESDIISSIALQVKDVVDLGVEVAIVVGGGNIWRGIAGSAKGIDRATADYMGMLA
TVMNSLAMQDALENMGVPTRVQSSISMQQVAEPYIRRRAIRHLEKGRVVIFAAGTGNPYFSTDTTAALRAAEIEAEVILM
AKNKVDGVYSADPFKDETAEKYETLTYLEVLNKNLGVMDSTASSLCMDNNIPLIVFSITENGNIKRVVLGEKIGTIVKGS
V
>lcl|NZ_CP026520.1_prot_WP_042233039.1_4201 [gene=tsf] [locus_tag=PC41400_RS21200] [protein=translation elongation factor Ts] [protein_id=WP_042233039.1] [location=complement(4717776..4718432)] [gbkey=CDS]
MAVTAAMVKELREKTGAGMLDCKKALDEANGDLTKASEILREKGLAAAANKAGRAATEGIVESYIHAGGRIGVLVEVNCE
TDFVAKTDNFRDFVKGIAMQIAAANPSYVRREEVPTEELDKEKEILRNQALNEGKPANIVDKMVEGRINKYYEEICLMEQ
SYIKDPDKTVSVLLNEMISTIGENISIRRFARYELGEGLEKKVDNFVEEVMAQAKGNA
>lcl|NZ_CP026520.1_prot_WP_009676777.1_4202 [gene=rpsB] [locus_tag=PC41400_RS21205] [protein=30S ribosomal protein S2] [protein_id=WP_009676777.1] [location=complement(4718534..4719232)] [gbkey=CDS]
MAVISMKQLLEAGVHFGHQTRRWNPKMDKYIFTERNGIYIIDLQKTVKKVEEAYNFVKSVSAENGTIMFVGTKKQAQDSV
KEEAERCGMFYINQRWLGGTLTNFQTIQKRINRLKELEKMEQDGTFELLPKKEVIILNKEKDRLDKFLGGIKGMKGLPSA
LFIIDPRKERIAVAEARKLGIPIVGIVDTNCDPDEIDYVIPGNDDAIRAVKLLTGKMADAVIEANQGEQTTA
>lcl|NZ_CP026520.1_prot_WP_042233037.1_4203 [locus_tag=PC41400_RS21210] [protein=hypothetical protein] [protein_id=WP_042233037.1] [location=complement(4719523..4720035)] [gbkey=CDS]
MKVNKALLAGIGIGMIVGSALLQLMITAAQPVQPLSQNSAVSIEQMDMNKLKTEAGKYYQVFDKNQPVFTKEQHEQELQK
QLEAEKAKTPSTQPAPSAQPQITALYISPGMRSSQVAKIVFVSGLVESQEQFEQAMAAKVATGKIKSGVHKFTGKPTLDE
IIQNLTSKPE
>lcl|NZ_CP026520.1_prot_WP_042233035.1_4204 [locus_tag=PC41400_RS21215] [protein=hypothetical protein] [protein_id=WP_042233035.1] [location=complement(4720022..4720666)] [gbkey=CDS]
MEPWQMIVLLGLVIVVFAGVIPRSAAKKGGQAPSAQLVKELEESMAHFASDLEQQNQALIDLFKETQREHKLEVAKLTAR
LEVLERRSADGNGAAVHGGRESESELSRHGSRSTEPRVYGEPAVQTHGSALSTPRPSQTEEFPAEAAAASVQPPQEAPRT
EPDLSVRARYAALFELHDQGRSTEQIAKQLGMNKGEVMLIMQLAKREGHSDESK
>lcl|NZ_CP026520.1_prot_WP_042233033.1_4205 [locus_tag=PC41400_RS21220] [protein=hypothetical protein] [protein_id=WP_042233033.1] [location=complement(4720698..4721018)] [gbkey=CDS]
MDMKPVDLQFAVHKNDQAGIRQQQLNHKPEMDQVLLGAENEKNAEHNRKASEKIGESEEAQIRKEGRQNDGQGGRGRQQR
KAPPSQEAEPPKAVHPFKGSHIDLSL
>lcl|NZ_CP026520.1_prot_WP_042233032.1_4206 [locus_tag=PC41400_RS21225] [protein=FapA family protein] [protein_id=WP_042233032.1] [location=complement(4721070..4722482)] [gbkey=CDS]
MDRILELDQIIRISVTDDKLTSLLKFADVGDDFTVTFDELQGLARANGIVHGLDTEALLRIAIDPQPYLLAETVIAQGTR
PVNGTDGYVDYLYGRNNDTHKPVETEDGRVDLKEVSTLNNIRKGQIIGKKVSPTEGTPGKAVTGEPLPAREGKEARFKLG
KNVVTDNEEKAIYAAIDGLVTYTDRSKVNVFPVYEVNGDVDYRTGNIDFVGTVVIRGNVSPGFKIKASGDIRIVGGVEAA
ELEASGSVEITAGIVGRNKAVIRAGHNVKSSFIQDATVEAADTVTVSQSILHSTIRAGKTVACNGPKGLIVGGTVQAGEK
VIARTIGNSLSNATAIEVGVLPELRNELIQLRSQLKALAENQSKTDKALILLDQLAASGQLGPDKVALRIKLNHTKRQSG
EEQRELRERIFEMEKSLEDSDRSSVEVISTVYGGAKIVIGRYTKFIKDPMSHVVFQLVAGDIGHSAYMPK
>lcl|NZ_CP026520.1_prot_WP_009676771.1_4207 [locus_tag=PC41400_RS21230] [protein=FliA/WhiG family RNA polymerase sigma factor] [protein_id=WP_009676771.1] [location=complement(4722512..4723294)] [gbkey=CDS]
MIDKKPNLAHFDTWRQWKQGGSREAKQTLIETYLPLVEYVSGRLAIGLPKNVSKEDLGSFGIMGLIDAVEKFDYERGLQF
ETYASWRIRGAIIDGLRQGDWVPRSVREKAKKIEEAYQKLEQKYLRTVTDEEISDYLQVTEDEFQHMLQEIAVTSICSID
EPIREEESETRLSLLVDERAKNPEFTVNEFYLKETLARAIEKLTEKERIVVSLFYFEELSLSEIAEVMSLSPSRISQLHS
KAMLRLRGVLSRYKVQLFEK
>lcl|NZ_CP026520.1_prot_WP_042233029.1_4208 [locus_tag=PC41400_RS21235] [protein=hypothetical protein] [protein_id=WP_042233029.1] [location=complement(4723310..4723726)] [gbkey=CDS]
MAGTIRWNFIVGAVSFVLTFLLSLSSNVWLTTLLRSIYSFIILFVVTFGFRFIMGLLLSANSTAVNETAAAAAEGPSPNK
NLGQHLDLTTPDETAQKAADGVDREEAEFAPLLPTKLVTRKQPDTEQSVRALRQMSEE
>lcl|NZ_CP026520.1_prot_WP_042233027.1_4209 [locus_tag=PC41400_RS21240] [protein=chemotaxis protein CheD] [protein_id=WP_042233027.1] [location=complement(4723729..4724223)] [gbkey=CDS]
MNQQTIIKVGMADLNVGMPGGVLKTTGLGSCVGLTLFDDFAKVGGMAHVMLPTSEIAREGTVNIAKYADTAIPELIDRMT
RLGASINRMEAKLAGGAQMFAFAGGGDTMRIGPRNVEACKEMLIRYKIRVKGEDTGQNYGRTIELNCDTGVLTIRSVQHG
LKDL
>lcl|NZ_CP026520.1_prot_WP_042233025.1_4210 [locus_tag=PC41400_RS21245] [protein=chemotaxis protein CheC] [protein_id=WP_042233025.1] [location=complement(4724201..4724839)] [gbkey=CDS]
MDTFRQLADYQLDVLKEIGNIGAGHAATALSTLLDKPVDMLVPKVMVLPFEEIPEIVGGAEEVVLAIFLRVVGDAPGNMF
FILNLESAKRMLFSLIGMELSDSGEYTELEISALNEIGNILAGSYLSSLADFTRLSMFPTVPSLAIDMAGAILSYGLLQF
GEMGDQALLIDTKFLEGNEEVQGNFFLIPDPESFGKLFAALGVEAHESTDDH
>lcl|NZ_CP026520.1_prot_WP_042233023.1_4211 [locus_tag=PC41400_RS21250] [protein=chemotaxis protein CheW] [protein_id=WP_042233023.1] [location=complement(4724844..4725305)] [gbkey=CDS]
MAEDKKIIVFALAHEEYGVEVDKVKTIERMQPMTRVPKTPAFVKGVINLRGVVVPVIDLRSRFGLPETEYSDNTRVIIVA
VNELEVGLIVDSANDVIDLNEDNIENPPEVVGGIKAKYLHGIAKIGEDRLLVLMNLEQVLNRDEIIQLEKIEA
>lcl|NZ_CP026520.1_prot_WP_042233020.1_4212 [locus_tag=PC41400_RS21255] [protein=chemotaxis protein CheA] [protein_id=WP_042233020.1] [location=complement(4725329..4727401)] [gbkey=CDS]
MELNQYLSMFIDESKEHLQALNDNMLSLENSPDNIDIVHNIFRSAHTLKGMSATMGFEDLASLTHEMENVLDLVRNLKLK
MDPFIFDSLFQSLDALDSMVEDIVQGGTGKADVTAIVTALKSIVSGDYLKSAPTAGGAGAESGEESVIALDQFQLSVLQQ
SIDANMQVFYVEVTVREDCVLKAARAYMVFEILERSGEVIKATPSVQEIEQEKFDRTFSVYYIANVGKADLEKEISNVSE
IEHVQIIVLDRETLDEMTGSGAAQAEVRAGEAQAAAAAAPAPAAKAAPAKETGGAAKPAAAAGGGPVGTRTIRVDIERLD
SLMNLFSELLIDRVRLEQLASEIKRNELTETVEHMARVSSDLQNIVLKLRMVPVDTVFNRFPRMIRDIAKSLDKKVDLVI
TGAETELDRTVIDEIGDPLVHLLRNAVDHGLETVTQRVAKGKSESGTVHLRAYHSGNNVFIEIEDNGNGINREKVLATAV
KKGIVAPEMADKMSNSDIDQLLFASGFSTADVISDISGRGVGLDVVKTKIHSLGGHIHVESNPGSGSIFSIKLPLTLSII
SAMLVKIGSEKYAIPLSSIVETALIQQAQIRSVQGNRMVDFRGSMIPLVSLSSVLDIPDYSEADETETEVVVVRKGEKMA
ALMVEEFIGQQEIVLKTLGKYFTKLFAISGATILGDGQVALILDTNELIK
>lcl|NZ_CP026520.1_prot_WP_042233018.1_4213 [locus_tag=PC41400_RS21260] [protein=chemotaxis response regulator protein-glutamate methylesterase] [protein_id=WP_042233018.1] [location=complement(4727438..4728604)] [gbkey=CDS]
MEPYRILVVDDSVFMRKIITDLIEEDPNFRVIAVAKNGAEAVVKTKELKPHAVTMDIEMPEMNGLEALERIMAATPTPVI
MLSSLTMSGAVETIRALELGAIDFIRKPSGSISLDLFKVKQLLHEKLRIAVKTRMQPQLPIEPAKRPERSVQPSAPFVKP
PSAAEIPKRITAAVQETGKPLSGVGSGSSKGFRDLVAIGTSTGGPRALQKVLSGLPADFPAPVLIVQHMPPNFTKSLSQR
LNDQCAVRVVEAEEAQVLEPGTAYVAPGGLHMTVVKADDKKYRISLSKSEPVSGHRPSVDKLFQSLLPLKELTRHAVIMT
GMGSDGARTIGELRKDGAATTIAESEETCVVYGMPRVAVENGAAMFVLPLGDIAPKLVERVTDKNGSR
>lcl|NZ_CP026520.1_prot_WP_042233016.1_4214 [locus_tag=PC41400_RS21265] [protein=MinD/ParA family protein] [protein_id=WP_042233016.1] [location=complement(4728641..4729537)] [gbkey=CDS]
MSDQAQELRKLIRLRHEKDTPSPGSALTGQPERKTRIITVTSGKGGVGKSNFTLNFALCLQARGYKTLIFDADIGFANID
VLMGISSKANLLHLIKGEKTIGELVQTGPKGLQFIAGGSGLYDLVRLSDSEADSFTDQIQELGGMADFILFDTGAGLSKE
TQRFIAAAQETLVVTTPEPTSITDAYALIKMVRSEDPDVTFKLVVNRAAHSREAKETAEKIAMVARRFLSVELQTLGFVE
DDQHVSRAVKEQTPFTIAYPNSPAAKSMNVLVDQFLGQQPKADSTGGFKGFLSRLIGR
>lcl|NZ_CP026520.1_prot_WP_042233015.1_4215 [gene=flhF] [locus_tag=PC41400_RS21270] [protein=flagellar biosynthesis protein FlhF] [protein_id=WP_042233015.1] [location=complement(4729530..4731305)] [gbkey=CDS]
MRVKRYIVDSMPDALHKIRTDLGKDAVILSTKEVKTGGFLGLFGKKRIEVIAALEEGQTGSGKSGPEGGAQARPSSKAAA
AAAVTAVAKARQEEAEKAAEQPRPNKEQELRQKEFAELLLRQSALQNEAVPTYGTGAAETLAARETSLLKKEETSDVMSG
SEPSGVTSNGQSLSGVREAGSDQRQAPGQQFPQPSPAVTDRPSGRPDAEEALPVIPQRPPAAAAAAYAQARPAAAEQRQS
LSAQAAPPVPPPAEAPAEPAAEAPQSSGFADEQAEALRAALAASRAEAAAEAAAAQSAAREDLMLAELQHLKAMMAGLAR
PSAQPQLPPVWGRLEEHLLAQEVQPDLVRQLIESARRDLAASADEAGARQAAEAVRKRMLALFPPPQGKPVSADSKIVHI
VGPTGVGKTTTIAKLAADQVLKQRRKVGLITSDTYRIAAIEQLRTYANILDVPVEVVFSPADLTRALHNLREADVIFMDT
AGRNFRNEMYVSELNTLIRHQGPSETILVLSLTSKYSDMKAITANFSKFKVDKVLFTKMDETYSYGAMLNLLDEFPIRLS
YVTDGQNVPDDIMVADENRIVDVVLEGYSYE
>lcl|NZ_CP026520.1_prot_WP_042233013.1_4216 [gene=flhA] [locus_tag=PC41400_RS21275] [protein=flagellar biosynthesis protein FlhA] [protein_id=WP_042233013.1] [location=complement(4731302..4733335)] [gbkey=CDS]
MKPKDLVVLIGVIGIILMMVVPIPTALLDFLLIINITISLMIILVGMNTQEALQFSIFPALLLVTTLFRLALNVSTARNI
LAHAEAGAVVETFGNFVAAGNIVVGFIVFLILVIVQFIVITKGSERVAEVAARFTLDAMPGKQMSIDADLNAGMINEQQA
RERRRKVEREADFYGAMDGASKFVKGDAIAGIIILLINLIGGLIIGISMKGMSFGEAGHTYSILTIGDGLVSQIPALLIS
TASGIIVTRASSEGTFSNDIVGQLTAYPKLLYIVAGTIAFLGIFTPIHWYTTIPISVILVFAARKMQQNLNRKQVEEEQM
VEEQQIEEVRSPESVISLLQVDPIEFEFGYGLIPLADAQQGGDLLDRIILIRRQCALELGIVVPVIRIRDNIQLRPNEYI
IKIKGNTVAGGELLLNHYLAMSPGFEDESVTGIETTEPAFGLPALWIDEMTKERAELSGYTVVDPPSVVATHLTEIIKKH
AHELLGRQETKTLIDNVREAYPALVDDLIPNVLSIGDVQKVLVKLLREKISVRDLVTILETLADYGPYSKDPDILTEYVR
QSLSRQITLQYSSPQDSLRVITVGPGLEKKIADSVQQTDQGSYLAIDPSSSQVIYHKLGEQVSKMVQSGHQPVILASPTI
RMYLRQLLERTMQDIPVLSYSELEPSVEIQSMGVVNL
>lcl|NZ_CP026520.1_prot_WP_042233011.1_4217 [gene=flhB] [locus_tag=PC41400_RS21280] [protein=flagellar biosynthesis protein FlhB] [protein_id=WP_042233011.1] [location=complement(4733351..4734448)] [gbkey=CDS]
MEPLRLKLDLQLFSQEKTEPATPKKRQDARKKGQVAKSMELPAAFILFFSFLSFYMIGGFMKERLIDLFSGTLRHDLLMD
VTMANIRMLFYQLLIQFLIMLAPILILTMVIALAGNYLQIGILFTGDPLKIKLSKLNPIEGFKKIFSMRSAVDFLKSMLK
MSVIGIVVYTTLWDEKAQLLSLSHVPLEKSLHYIWSLTLTLGLKIAFILVVLALFDYLYQKYEFEKSIRMSKQDIKDEYK
KMEGDPLIKGKIREKQRRMAMQRMMQEIPTADVIITNPTHFAVAIRYDSANMQAPKVVAKGTDHVALKIREIAKEHGIMI
MENKPLARALYAQVDLEESIPGDLFQAVAEVLAYVYKIKGKVKNA
>lcl|NZ_CP026520.1_prot_WP_042233009.1_4218 [gene=fliR] [locus_tag=PC41400_RS21285] [protein=flagellar biosynthetic protein FliR] [protein_id=WP_042233009.1] [location=complement(4734477..4735274)] [gbkey=CDS]
MELWFQLLPNAMLIFCRMTSFLVVVPLFSSRNVPSMFKIGLALFLTLIVTAAQGTSQLVPMDAQYVLLVIREIIVGLLLG
FIAYLFFTAVQIAGSFIDMQIGLGMANVIDPMSGTTVPLMGNLKYMLAMLLLLSFNGHHFMLQGIMNSYDWIPLSNNLFA
QLYGGQITEFLVKTFSSVFSLAMQLAAPVVAAMFLTDLGLGLLTRVAPQFNIFVVGVPLKIIIGFLLIALVMPELIGLFH
QLFDRMFEAMQKLLNIVAGKALEAP
>lcl|NZ_CP026520.1_prot_WP_009676755.1_4219 [gene=fliQ] [locus_tag=PC41400_RS21290] [protein=flagellar biosynthesis protein FliQ] [protein_id=WP_009676755.1] [location=complement(4735281..4735550)] [gbkey=CDS]
MSSQFIIKLAGEAVYTVLKASAPMLVIALAVGLIISIFQATTQIQEQTLAFVPKIVAVLLSVLIFGPWILNTLVDFTFNL
FNNLHQYIG
>lcl|NZ_CP026520.1_prot_WP_042233007.1_4220 [gene=fliP] [locus_tag=PC41400_RS21295] [protein=flagellar type III secretion system pore protein FliP] [protein_id=WP_042233007.1] [location=complement(4735576..4736352)] [gbkey=CDS]
MSKNKGTLIILLALLMFIGWAGNTAYAAPDPQNPIPGISFNVDSGSGASGASSTVSVILLLTVLSLAPSFLILMTSFTRI
VIVLGFVRTSLGTQQMPPNQVLIGLALFLTLFIMAPTLGQINDTAVQPYMKGELNQTQALQKASVPIKQFMWNHTRPKDL
KLFLDYTKAEAPKGVEDTPLTALVPAYAISELKTAFQMGFMIFIPFLVIDMVIASTLMAMGMMMLPPVMISLPFKILLFI
LVDGWYLVVKSLLLSYST
>lcl|NZ_CP026520.1_prot_WP_042233005.1_4221 [gene=fliO] [locus_tag=PC41400_RS21300] [protein=flagellar biosynthetic protein FliO] [protein_id=WP_042233005.1] [location=complement(4736339..4736977)] [gbkey=CDS]
MKKQAGKLLLATAGLIVQLPFWTSVCFGAPEPGENIVPGSGETGSIDSIVMIGKVIFFLLLIIGLFYGIMKMLSKKSGFR
FGRAMQSLGGVPLGQNKSIQVVEIGDSLYIVGVGDNIQLLEKIADADEAARLKEQITAQTRAATEFITVGKWLEKIRKKP
GENNVVEEEELSPNFHQVFQDKLQNVAQRNKSVEEWLQEENEQDRLKSNEQK
>lcl|NZ_CP026520.1_prot_WP_009676759.1_4222 [locus_tag=PC41400_RS21305] [protein=response regulator] [protein_id=WP_009676759.1] [location=complement(4737012..4737374)] [gbkey=CDS]
MGNRILIVDDAAFMRMMIRDILTKNGYEVVGEGNDGVQAVEKFKELRPDLVTMDITMPEMDGINALKEIKKIDPNAKVIM
CSAMGQQAMVIDAIQAGAKDFIVKPFQADRVIEAIKKTMG
>lcl|NZ_CP026520.1_prot_WP_042233003.1_4223 [gene=fliY] [locus_tag=PC41400_RS21310] [protein=flagellar motor switch phosphatase FliY] [protein_id=WP_042233003.1] [location=complement(4737401..4738684)] [gbkey=CDS]
MTSKDYLSQEEIDALLKQAVDERSDDNPSSSHSSNIDDYLSSLEQDALGEIGNITFGSAATALSTLLGKKVDITTPKVSC
IPRVQLVNEFPQPHVAVHVNYVDGFDGINLLVIKTRDAQVIADLMMGGDGHVEGDELSEIHVSAVQEAMNQMMGSSATSM
STIFNRFVNISPPGIDILNLSNGEGKLPNEDVFIKISFRLTIGDLIDSTIMQLLPVDFSKEMVHILMGGGSEEVYEEAAA
AVSEEVPPAGTGQVQADTNAHAYQQPVPEPPYTNPAEAPYTNPSMGGNPMPMNQPPAQQSYAQQPMNHAPQGGNYGGMNR
NVNVQPVQFSNLQQGGYNQADDTNLNLLLDIPLKVTVELGRTQKVIKDILELSQGSIIELDKLAGEPVDILVNNKLIAKG
EVVVIDENFGVRVTDIVSQWDRIQKIQ
>lcl|NZ_CP026520.1_prot_WP_042233000.1_4224 [gene=fliM] [locus_tag=PC41400_RS21315] [protein=flagellar motor switch protein FliM] [protein_id=WP_042233000.1] [location=complement(4738674..4739672)] [gbkey=CDS]
MVDVMSQNEIDALLAALSSGEMDAEELKKEDTQRKVRLYDFKRAVRFSKDHIRSLTRIHENFARYLTTYFSAQLRTFVQI
NVVQVEQLPYDEFIRSIPKMTILNIFEAEPLEGRMVLEVHPNVAYAMLDRLLGGVGVSPQKIAALTEIETIVMERIFSRA
FESLQEAWKTIIDLSPRLEALETNPQFMQIVSPNETIALISLSTKIGDTTGMINLCIPHVVIEPIMPRLSVHHWFVSQKK
SRAPEEVEILQDRLTQAKLPIIAELGESQITIHEFLNLQPGDVITLNKPVNEPLNIKVGEKLKFTGSAGTTKSKMAVQIG
EIVNEGAEVHDE
>lcl|NZ_CP026520.1_prot_WP_042232998.1_4225 [locus_tag=PC41400_RS21320] [protein=flagellar basal body-associated FliL family protein] [protein_id=WP_042232998.1] [location=complement(4739714..4740157)] [gbkey=CDS]
MLKNRFFLMLVSILVAITLVLTAAFVLWTVMDNKPEAGAEKTQTSAEIKENTVIMKDITTNLASQGSFVKISLAFELENK
KAKTDFENLIDSRIKGTIIRTLSDMNSDQISGSKGQDNLTNTLISKLNPLLNEGKIRRINITDIVLE
>lcl|NZ_CP026520.1_prot_WP_009676773.1_4226 [locus_tag=PC41400_RS21325] [protein=flagellar FlbD family protein] [protein_id=WP_009676773.1] [location=complement(4740157..4740378)] [gbkey=CDS]
MVRLTRMNGKPLVINALMIEMMEETPDTVITLINGKKIMVLEQEAVVTEMVEQFLKRIGAVQATIKSANTEGL
>lcl|NZ_CP026520.1_prot_WP_042232996.1_4227 [gene=flgG] [locus_tag=PC41400_RS21330] [protein=flagellar basal body rod protein FlgG] [protein_id=WP_042232996.1] [location=complement(4740441..4741271)] [gbkey=CDS]
MLRSMYSGISGMRGFQTKLDVIGNNISNVNTVGFKKGRVMFQDILSQTVAGETAPTSTTQGGINSKQIGLGVGTAAIDTV
HTGGSAMTTNVPTDLRIDGNGFFGVTQNGQDPLYLTRAGNFYVDGDNYIVNPDGMYLMSTDSEPIQLDANATKFSISQDG
TVMQVIGGVPTPSDYKVAVVKVANPAGLNKAGGNLYEYTPNADVNNTIDVAYLSNALPADPVTGTGSIIAGQLEMSNVDL
TEEFSEMIVSQRGFQANSRIITTSDEILQEIVNLKR
>lcl|NZ_CP026520.1_prot_WP_009676749.1_4228 [locus_tag=PC41400_RS21335] [protein=TIGR02530 family flagellar biosynthesis protein] [protein_id=WP_009676749.1] [location=complement(4741392..4741766)] [gbkey=CDS]
MTERITVGRLYPTPPAPLQRNGGARPAAGRSEFDQLLQSELVKFSHHAETRLKQRGIQIDNEQLAKLENAIDKAAAKGAK
DTLMVLGGTALIVNVPNRTVITALDGAKLQDQVFTQIDSAVIIS
>lcl|NZ_CP026520.1_prot_WP_042232992.1_4229 [locus_tag=PC41400_RS21340] [protein=flagellar hook capping FlgD N-terminal domain-containing protein] [protein_id=WP_042232992.1] [location=complement(4741763..4742224)] [gbkey=CDS]
MAELWGDLTKTGSTTNTASAASKAKDKKDGSILGKDDFLKILITQLQNQDPMQPMQDKEFVAQMAQFTSVEQLSNMASEM
KLLRSSLGFTSSLIGKDISWMGKTDSGETVMSNGVVESIIVRDGVQYATVGGKEVKLEEVLKIENQKPAVPAP
>lcl|NZ_CP026520.1_prot_WP_042232990.1_4230 [locus_tag=PC41400_RS21345] [protein=flagellar hook-length control protein FliK] [protein_id=WP_042232990.1] [location=complement(4742269..4743666)] [gbkey=CDS]
MEMPVASTTPSAPSAGSTAVKAGTAKTAAPAATSAGNGKGDAGTFQQVMNGQLQKEAGEENASPVDSILAGLLQVIPFLP
APVQEALSDMEGQTGTDGMADLLLSSIQSSPELFSQLSQSDAMQQWANDASALLQLLSGTQQTGLQAIQLDSGDASLKLQ
KTILDLTAMLKQQPDHPLLQQLSAEFKQLVESALPSLVNQQQADAAGKAAAKAAEMELTLPASATDSNPDAAAPVMTVTK
TGSVKASLELLAARTGVWNSVRQAAHESVEAVQTSTTSASENEPAANGLTGLQEGMKTTLPEQAAKAAAPTISAQNFAQE
LSQFMVKRMSLKMGNGFSEAKLSLRPQHLGTIDVVLTMQNGQLVAQFTADTALGKELLESQLAHLRQSLHSQGVQVDKME
VGQNTMSSTMFQDSRQQQSSQQFLQQRGGGSNQGNSEELTEWNEELKILEETRKAALGNSFETSA
>lcl|NZ_CP026520.1_prot_WP_042232987.1_4231 [locus_tag=PC41400_RS21350] [protein=hypothetical protein] [protein_id=WP_042232987.1] [location=complement(4743697..4744599)] [gbkey=CDS]
MEEAGTESSSGGKLERFLVWFLIPFVFTAVLLGVLLSVFDYDVMKTVQRTLNKIPIVGEMIPLPKEDAQSAPAAKGTEED
AVKTKTKEIEQLNSKVQELKTELDKADQTSAQKDKTILDLKQQKAELEDKELAKQDEEYDKKVRELASVYAKMMPSKAAQ
IIQNLSPQETILVLNAMKTDERVKVLEKMDPKKAADASIALKDLTTSQDQKIAALQDRLVLQQKPDSAQKKMTSAELSQT
FASMDASKAAKVLTDMYGKNSAQVLTILREMDTSKRAALMGEIAGISKETAASITGNLVP
>lcl|NZ_CP026520.1_prot_WP_042232985.1_4232 [gene=fliJ] [locus_tag=PC41400_RS21355] [protein=flagellar export protein FliJ] [protein_id=WP_042232985.1] [location=complement(4744624..4745070)] [gbkey=CDS]
MKFQYAFQQIVNLKNNERSQAEWNLSTAIGNLNHEESSLQELIHLKNEAQEKMLETSMRPVSMLEIRQLQSYVDHIEHLI
ARKRHDVSKAQRHVNHNRDLLNDKMVDEKVWTKARDKALEQFTGFVLKKEQEALDEIASNRHIRLSYQ
>lcl|NZ_CP026520.1_prot_WP_042232983.1_4233 [gene=fliI] [locus_tag=PC41400_RS21360] [protein=flagellar protein export ATPase FliI] [protein_id=WP_042232983.1] [location=complement(4745078..4746403)] [gbkey=CDS]
MSQGSPNAGKYVDFLRQLDPVRVNGKVTQVIGLTVESEGPDVSIGELCHIYPHKSNQPLMAEVVGFRGNKVILMPLGELE
AIGPGCDVVGTGKPLTVQVGHELLGKVLNGLGQPLDGSFLPSRMAHVSTNNVPGNPLSRPRVQEPLGVGVRAIDGLLTVG
KGQRVGIFAGSGVGKSTLLGMVARNTSADVNVIALIGERGREVLEFVEKDLGPEGLARSVVIVATSDQPAMVRIKGAFIA
TSIAEYFRDRGMNVMLMMDSVTRFAMAQREVGLAVGEPPATRGYTPSVFALLPKLLERSGTGLKGSITAFYTVLVDGDDM
NEPIADAVRGILDGHIVLDRNIANKGQYPAINVLASVSRVMKEIVTQEQLDAAAHLKRMLSVYKDSEDLINLGAYQRGSN
QEIDISLQYMDEILDFVKQKTNEKVTLEEAQQRLIDHFSLG
>lcl|NZ_CP026520.1_prot_WP_042232981.1_4234 [gene=fliH] [locus_tag=PC41400_RS21365] [protein=flagellar assembly protein FliH] [protein_id=WP_042232981.1] [location=complement(4746400..4747233)] [gbkey=CDS]
MSNILKPFHYAPADETNAYRVPVKPIYTPQAELPEDEPRFTPEEEAQLSEAESLKRHILEDAESFAQSHMQQAIQEVSEL
KEQTEAEIEAWWMQRREEDEQHQEQARTTGYETGYQEGLDQARQDVSQQYDEMLSEARQVLEQAYRVKQQVIQEAEPFLI
ELSCSIAEKIISKQLTVSPEWMIETIRSVLARRKESGTITLCIAPRHFAYIQDAREELLSAIDSQAELEIIPDASVQDEG
CVVRSTFGSIDARIDTQLHEIKLALQRIALHGEGDES
>lcl|NZ_CP026520.1_prot_WP_042232979.1_4235 [gene=fliG] [locus_tag=PC41400_RS21370] [protein=flagellar motor switch protein FliG] [protein_id=WP_042232979.1] [location=complement(4747226..4748245)] [gbkey=CDS]
MARGAIQQGLTGRQKAAILLISLGPEVSAQIFKHLREDEIEQLTLEIANVRKVDALEKESILSEFHQICRAQEYISQGGI
AYAKEILEKALGSNKAVDIINRLTANLQVRPFDFARKADPAQILNFIQNENSQTIALVLSYLQAEQASIILSSLPQEKQA
DVAKRIALMDSTSPEVISQVERVLEQKLSATVTQDYTSAGGISSIVQILNGVDRGTERTILDSLEIQDPELAEEIKKRMF
VFEDIVNVDNRSIQRIVRDIENADLQLALKVASEEVREAIFRNMSKRMAETFKEEMEFMGPVRLRDVEEAQTRIVSTIRR
LEESGEIIIARGGGDDIIV
>lcl|NZ_CP026520.1_prot_WP_042232976.1_4236 [gene=fliF] [locus_tag=PC41400_RS21375] [protein=flagellar basal-body MS-ring/collar protein FliF] [protein_id=WP_042232976.1] [location=complement(4748258..4749835)] [gbkey=CDS]
MNERLLQYWEKAKQYWNQYSKTTKLTFIAALFVLLLTIGIISYNLAKTEYSLAYTNLQPADAAAIKGYLDTAKIPYNLSS
DGKSIEVPKNEVANVKLGIESQGLNKGSSIGYDAFASDSAFGMTDNEFKVKYLSAVQGELQQLINSNEAVDKSKVLISMP
DETVFLANQKQEATASVVLNVKQGYSLDQGQVDTIYNLVSHSVKNLPLQNITISDQYGQFLAYSKETNGGQSTSGLVQNQ
FDIRRKFETSVQGNITNLLNKMLGPDKVVVSVFSTMNFDQKKTTENLVTPPNLEDKKGLEISTQESSKTSTSTGGNPGGV
AGTGQTDVPGYPGGAASGNANSEESSKTVNYEINRITNDIIQTPYVVKDISINIGFDSTGKDQAAVAELKGLIEGQMANI
VRASLADNKVEYTPAQIQDKVKVVPQTFVTPSGIAGTNWQQYLLYGGIGLAALLLGALGMYAIRRKKKAGEEEDELDLPP
AKVEYPTIDIENVSNENQVRKQLETLAKKKPDEFVNLLRTWLVDE
>lcl|NZ_CP026520.1_prot_WP_231392005.1_4237 [gene=fliE] [locus_tag=PC41400_RS21380] [protein=flagellar hook-basal body complex protein FliE] [protein_id=WP_231392005.1] [location=complement(4749885..4750202)] [gbkey=CDS]
MSVVNPTYLNAFSPVQTIKQPVGSEPASASQFTKQFSSFLSEAMNGLNSDQHTVDGLNQQFAKGELSDVHQLMIASEKAS
LGLEFTVQVRNKVVEAYQEVMRMQL
>lcl|NZ_CP026520.1_prot_WP_042232974.1_4238 [gene=flgC] [locus_tag=PC41400_RS21385] [protein=flagellar basal body rod protein FlgC] [protein_id=WP_042232974.1] [location=complement(4750222..4750671)] [gbkey=CDS]
MRITNSFDISASALTTQRLRMDVISSNIANADTTRASFVNGQWTPYQRKIVSVEPKNQPTFAQMLNSAMTDGTGQGVRVS
SISGDTTPFKQVYNPSHPDANQQGYVMMPNVDVAKEMVDMISATRSYEANVTALNASKGMISKALEIGR
>lcl|NZ_CP026520.1_prot_WP_016423454.1_4239 [gene=flgB] [locus_tag=PC41400_RS21390] [protein=flagellar basal body rod protein FlgB] [protein_id=WP_016423454.1] [location=complement(4750675..4751088)] [gbkey=CDS]
MFLLNQAGFQWMEKSLDASALRQRVISNNVANADTPHFKRSDVLFEELLQEKMTGNQPVLEGRRTNSRHLQIPSNTNLPG
QAVVESDPNSLMNNNLNNVDMDYEMSLMAKNQLKYNTMVQEMNNEFRKLKIAMDGRR
>lcl|NZ_CP026520.1_prot_WP_042232971.1_4240 [gene=codY] [locus_tag=PC41400_RS21395] [protein=GTP-sensing pleiotropic transcriptional regulator CodY] [protein_id=WP_042232971.1] [location=complement(4751309..4752094)] [gbkey=CDS]
MTLLNKTRRLNRLLQKEAVSAVSFMDMAEVLRDTIHSNVYVVSRRGKLLGYATTNELPTSAINRIVAEEKRFSPEFNHFL
LKVDETTSNSDALEMFPTFVEEMKNLREYTYFTIVPILGGGDRLGTLVLCRTHSAFTDDDYILAEYGSTVVAMEILHMRA
EEVEQEARSKAVVQVAIGSLSFSELEAVEHIFEELEGKEGLLVASKIADRVGITRSVIVNALRKLESAGVIETRSLGMKG
TYIRILNDQLIQAIEQVKSKM
>lcl|NZ_CP026520.1_prot_WP_042232969.1_4241 [gene=hslU] [locus_tag=PC41400_RS21400] [protein=ATP-dependent protease ATPase subunit HslU] [protein_id=WP_042232969.1] [location=complement(4752158..4753558)] [gbkey=CDS]
MSTSSLTPRQIVQELDRYIVGQKQAKKSVAVALRNRFRRSRLEESMRDEIVPKNILMIGPTGVGKTEIARRLAKLVGAPF
VKLEATKFTEVGYVGRDVESMVRDLVETSIRMVKSERTESVKERAEELANERLVSILAPSASRPKFQRNPLEMIFGNQQG
SQEPEEDDKDPQLATKRAQIKEQLKRGELENYIVEVDVEDNSPSMLDMLAGQGQDQAGMNMQEMFGSLLPKRTKKRKLPV
KEARKVLTQDEAQKLIDMDEVIQESIRRAEQSGIIFLDEIDKIAGNGRGSGGPDVSREGVQRDILPIVEGSTVVTKYGPV
KTDYILFIAAGAFHVSKPSDLIPELQGRFPIRVELSNLTLEDFVSILKEPKNALTKQYTALLETEGIRIEFSQSAIEEIA
RIAVEVNQSTENIGARRLHTILEKLLEDLSFEAPDLNLEEMVITPEYVREKLGDIVQNRDLSQYIL
>lcl|NZ_CP026520.1_prot_WP_009674471.1_4242 [gene=hslV] [locus_tag=PC41400_RS21405] [protein=ATP-dependent protease subunit HslV] [protein_id=WP_009674471.1] [location=complement(4753635..4754186)] [gbkey=CDS]
MDPTNATFHATTIFAIRKDGRGAIAGDGQVTFGNSMIMKNTAKKVRRLYRGKVISGFAGSVADAITLFEKFEGKLEEHHG
NLQRAAVELAKEWRSDRVLRKLEAMMIVMDEHGLLLISGNGEIIEPDDGIVAIGSGGSFALAAGRALSRYATDMSAADIA
KASLEMAADICVFTNHNIIVEEI
>lcl|NZ_CP026520.1_prot_WP_042232967.1_4243 [gene=trmFO] [locus_tag=PC41400_RS21410] [protein=FADH(2)-oxidizing methylenetetrahydrofolate--tRNA-(uracil(54)-C(5))-methyltransferase TrmFO] [protein_id=WP_042232967.1] [location=complement(4754346..4755680)] [gbkey=CDS]
MTAGQKEKVTVIGAGLAGSEAAFQIASQGVPVVLYEMRPVRKTPAHITDKFAELVCSNSLRANGLTNAVGVLKQEMREMN
SLILSSADKHAVPAGGALAVDRDGFSGEITDTLRNHPLIEVRTEELKEIPDGITVVASGPLTSPDLSASLQALMGEEYLY
FYDAAAPIVEKDSIDMSKVYLASRYDKGEAAYLNCPMTEAEFDAFYEALITAETAPVKEFEKEIYFEGCMPIEVMAKRGK
QTALYGPMKPVGLINPHTGELPYAVVQLRQDNAAGTLYNLVGFQTHLKWGEQKRVLQMIPGLENAEFVRYGVMHRNTFIN
SPRLLKPTYQFKNRDKLFFAGQMTGVEGYVESAASGLIAGINAGRLAQGLDCLEFPLETALGSMAQYITTADTKHFQPMN
ANFGLFPPLGEKIRNKKMKYEKLAERAIETIQNFSQNTVQSTCK
>lcl|NZ_CP026520.1_prot_WP_042232965.1_4244 [gene=topA] [locus_tag=PC41400_RS21415] [protein=type I DNA topoisomerase] [protein_id=WP_042232965.1] [location=complement(4755717..4757813)] [gbkey=CDS]
MADSLVIVESPAKAKTIGKYLGSKYIVKASMGHIRDLPKSQIGVEVERDFEPKYITIRGKGSVLKELKDASKKVKKVYLA
ADPDREGEAIAWHLAHYLELNDTDSCRVVFNEITKQAVKDAFKTPRPINIDLVNAQQARRILDRLVGYKISPLLWKKVKK
GLSAGRVQSVAVKLIMDRENEIKAFVPEEYWTITAMLEAGKAHFEAKFHGMDGAKKELGSEEEVQAVLKAIEGAPFVVKE
VKEKERLRNPSPPFITSSLQQEAARKLNFRAAKTMSVAQQLYEGIDLGKEGTVGLITYMRTDSTRISPVAQEEAKEYILG
KYGESFYPETPRVYTKKGSNTQDAHEGIRPSSVTRDPDSVKEFLSRDQFRLYKLVWERFVASQMSSAVLDTMTIDLNVGN
AVFRAVGSKLKFPGFMKVYVEGNDDGTTEEDKLLPPLKSGDQVVSNEIQPKQHFTQPPPRYTEARLVRALEELGIGRPST
YAPTLETIQKRGYVAIEEKKFVPTELGELVIQLMSEFFPEILDVEFTAQMEEELDHVGEGKENWVKVLDEFYSSFEKRLE
FAEEEMKEIEIQDEVSDEICEKCGKHLVYKMGRFGKFLACSGFPDCRNTKPIVKDIGVTCPKCKEGKVVERRSKKGRIFY
GCDHYPECDFVSWDKPVAQPCPNCGSMMIEKRTKTSVNHQCVECGFKEEVKDEANDEG
>lcl|NZ_CP026520.1_prot_WP_042232964.1_4245 [locus_tag=PC41400_RS21420] [protein=YolD-like family protein] [protein_id=WP_042232964.1] [location=complement(4758252..4758584)] [gbkey=CDS]
MGKKLEGNGLYESSRMILPEHKEAIQRQLRDLNRRTKPILDEQAWEMMLLEIVDARSAGKEITLVIFEPFADKELRGKVT
KFDMHLRRVRIDYEDDWDWIKWDDIVSVHI
>lcl|NZ_CP026520.1_prot_WP_042233289.1_4246 [locus_tag=PC41400_RS21425] [protein=hypothetical protein] [protein_id=WP_042233289.1] [location=complement(4758598..4758852)] [gbkey=CDS]
MLSDTERKVLRILYNFSNIHHCMPDLSELETKSGYNQENIKQALSTLREQGYIMGKKLSEVKIIEGWPREKKEEKKTYIN
PWMD
>lcl|NZ_CP026520.1_prot_WP_042233286.1_4247 [locus_tag=PC41400_RS21430] [protein=helix-turn-helix transcriptional regulator] [protein_id=WP_042233286.1] [location=complement(4758954..4759187)] [gbkey=CDS]
MEVVPLPLKVHIKLDELIKERGISQRELSRLTGIRQPSINEMCNNTTQRLPLDNLGIICEVLKIDIADILVLVNESK
>lcl|NZ_CP026520.1_prot_WP_042232963.1_4248 [locus_tag=PC41400_RS21435] [protein=hypothetical protein] [protein_id=WP_042232963.1] [location=4759620..4759805] [gbkey=CDS]
MTIFQWVVWAAMKKWHVLFKDEKGKRGWMQTSDLDLWGHVGSLKVINKKLCEIIAIHQVSE
>lcl|NZ_CP026520.1_prot_WP_084706664.1_4249 [locus_tag=PC41400_RS21440] [protein=helix-turn-helix transcriptional regulator] [protein_id=WP_084706664.1] [location=4759934..4760107] [gbkey=CDS]
MTQAQLAQRLNLSDSFVSKVVRGEKRLSLIKSREAAIILNCYMDDLYEWIDIEEGKR
>lcl|NZ_CP026520.1_prot_WP_042232959.1_4250 [locus_tag=PC41400_RS21445] [protein=M15 family metallopeptidase] [protein_id=WP_042232959.1] [location=complement(4760354..4761043)] [gbkey=CDS]
MLTLDYVLKKSEFRLAGLHPNVAAASKRLIVNCYKRGVTIVITQGLRTFEEQDALYCQGRSKPGPKVTNAKAGESFHNFG
VAIDFALLMPDGKNVSWDMTMDSDGDKISDWMEVVTEAKQLGFSWGGDWTGFKDYPHFEMTFGLTIKQYQAGKRPAVTLI
QEEDMEVEVLVFEHDWMWKQLGDSLDGLYRKGVISDYTWAEKAYTRKLTKSELTWINIVISARNAGVNV
>lcl|NZ_CP026520.1_prot_WP_042233284.1_4251 [locus_tag=PC41400_RS21450] [protein=phage holin family protein] [protein_id=WP_042233284.1] [location=complement(4761030..4761308)] [gbkey=CDS]
MNWETVINLIDSSFIVVVAACWVVGYVLKSTPRFPDWLIPYAVGVFAITFTVWTEGFSPANVLQGVLCAAAAVYGDQLTK
QVKKGVGKNADS
>lcl|NZ_CP026520.1_prot_WP_241688714.1_4252 [locus_tag=PC41400_RS21455] [protein=hemolysin XhlA family protein] [protein_id=WP_241688714.1] [location=complement(4761325..4761624)] [gbkey=CDS]
MGAGGEHQMEDMKKMHDTMVDVRLELRTISAKVDILTDASRKIEEIEDVAKGAMDSTKTAHKRLDKIDKFITWFTTTVIG
AIILAIMGFLIRGGFNLPK
>lcl|NZ_CP026520.1_prot_WP_042232958.1_4253 [locus_tag=PC41400_RS21460] [protein=hypothetical protein] [protein_id=WP_042232958.1] [location=complement(4761631..4761834)] [gbkey=CDS]
MAMAIVKTKLEILVDLSDPVEEIKSCIVAIVNQHPAKQKEIMLQVDTWVGEALAQMSLHESGEEKEG
>lcl|NZ_CP026520.1_prot_WP_042232956.1_4254 [locus_tag=PC41400_RS21465] [protein=hypothetical protein] [protein_id=WP_042232956.1] [location=complement(4761894..4762580)] [gbkey=CDS]
MIILFLSFFTLIGCLDKTVTTNQELVPENVSKTPDPDIDTEIFHLNDKEIQEAIELGKSLNRDSYKSFIEKNYALESTSD
NLGSLSKNKPVAMISTPYEEIVSDSYNKSQKFEKYDVEDAKYRLLLGGSIFFYFKTYGDEIRDFDNLHAVLKQDESIVQP
YKGKLTSNSTVPELTKEKSPYPAYTKVMIAQFTDIKKNLDLSKPVELIFMLNGKEQSVTYSIEISKYK
>lcl|NZ_CP026520.1_prot_WP_042232954.1_4255 [locus_tag=PC41400_RS21470] [protein=hypothetical protein] [protein_id=WP_042232954.1] [location=complement(4762649..4763116)] [gbkey=CDS]
MMKKYLYLLFGIFIGLGLSLGIGAHAEVTSMIGKVVEGSFPVLINGEKSEKDALVVDGTSYLPVRSAGQLFGYDVSFIDS
QVILKAREGVVNLEKLRSLQQEQTQINDEEAAAKLKKWEQEKKEWDKWQEDLEKEREEHKKDLEEWLGLSDKKVE
>lcl|NZ_CP026520.1_prot_WP_053228770.1_4256 [locus_tag=PC41400_RS30115] [protein=phage tail protein] [protein_id=WP_053228770.1] [location=complement(4763185..4765512)] [gbkey=CDS]
MLKVFDQNLTPVGVLPDASDVERKRRLNSDYELSFLLPMTSPNYREKVRHKGHVQDERGQYYVINDRKRVRDGRKLTALI
ECSHIMFKLSDFKMPYTGYIEEAFGTNILTLTNAITAATGGKFLFSIDDQFDKKDIKDFGRGNALQALNQVVKTFEAEVE
PNNFTIHLKKKIGTDTGMQYRFKKNIVSNTFKDSTRALVTRLYAQMKDGLSFIGLSASELTTEEYNLLNTIPGAIVDGKI
RVNYLISPYAGYWSNTTNTYYDGELINQDIEDPKELLEATRKALKEQEVPSVEVSIEAADLFKLDNSEPRPNMGDPVYLI
DPEMELNNITARVVEITEYPYARDKHTQVTLANYLLRDYADIIADLDRAKQIVNDITSGAKLRTTVFEAFAKQAIYDINS
SKTEVKYDQRGIILEDKSNAKNQVVMSSNGVYLTTDGGVTPRAAITANGVVAEVIVGVLGQFVQLKANQIVVGSQGEKIE
DGLIKSAGEWNNKVGQNTPYNGVIINTTDGVSVIRSDNKTRVIMNATEGFKIQGNDGGGWKDKLFADLNGKLTAEDLVAN
RFIMKNGGDVLIDGNTRTIDFSKFTTKLGTISTANIETLIVGQNVIMGPNAYISWNQVAGRPNTTYIDANGIYTGKVTAN
QIDTNGLSAQRIFQAGSPQNYAQIGGSYGDFVMYYNNSELFRFYNGLGNIQFQFLGSSLLSFGRDAYGRDSANPLKIWDF
SLATVRGISVNAVEGLEARLRFLEDRAFIGTRREGGNLVFSSKNYSNVDYGNVVG
>lcl|NZ_CP026520.1_prot_WP_174446839.1_4257 [locus_tag=PC41400_RS21480] [protein=hypothetical protein] [protein_id=WP_174446839.1] [location=complement(4765519..4765800)] [gbkey=CDS]
MAVLSFLRNTYCRSIFVYGTDRFTPRDGCKGIREEYREPVKEYFAETYELSHLQAALESGWINQQEYEETAALFPEGNEE
ATPDQTSPPELKE
>lcl|NZ_CP026520.1_prot_WP_053228769.1_4258 [locus_tag=PC41400_RS21485] [protein=hypothetical protein] [protein_id=WP_053228769.1] [location=complement(4765802..4766263)] [gbkey=CDS]
MKEAIITDALGVFQETTLVADDVTGVFPIYGSPEEDNEEEVPESVITGYQIAVSIPPGLFTPKWDFEAEAWTEALSAEEI
EKLKNPPQPVTLQDRVSQLETENASLKASAARANADLQEYMEFSMKVQEEQKTQQARMNADFQELIELLTQEG
>lcl|NZ_CP026520.1_prot_WP_042232952.1_4259 [locus_tag=PC41400_RS21490] [protein=hypothetical protein] [protein_id=WP_042232952.1] [location=complement(4766275..4769925)] [gbkey=CDS]
MAFIPRIIEKFLDFANLKRHNDNFQDIANEFTALDGRIQSNTAAIDNRYTKAEADAKDAAVSSAALTALNTHKASGDHDA
RYYTKVNMQTSGGSLIHWENITDKPNFADARWKSPVKDKATLDALLVGNTDGDIRLVLADETVYEWDADTAGANKWRPIG
AMGNGLTSHSSLTNLSNDDHKQYHNDARGDARYYRKDEIDVQMAGKIQQNGKLTGDLDFSSREAKNLVVHRAAVEPAQPV
EGQLWYHTGKKAMYIYKGATLGWVDISGKGAVIRDQEFTALPGQTVFDITVGRYETNTNAITVYKKYVTTGTYELVPEAD
YTESSETSFTLIKAAAGGEAYYVKFFENSPEVINESVKRDGTLQVNLNAEMLNGRRSTDFASSIHGANHVTGGSDVIPNA
VSGGSSGLMSGADKLALDNIQKDLATSTSKSITLNKPVQVVTADRTSRLKLDRFKGRTLVNLVGRDGNCEDASRWIDYQT
SHALDTTNYVSGKSSLKVILSSGFTTGSAITANPVSFVASKYYLLAGWLKNGNANYMNLSVSGQGAATATNTATSTSAFT
FAYKAFTGVSTTSTGINVSVNGAAGQYGYADEVRVYELSKAEYDAISGMTTEDIDAKYPYVDAVQHTTNPYVIRLGENLV
PASDSWIVPVPTRSSITGPYSTTMQYNASENVYVEFFVPVVPGQQYTATVTAEPANASPYYYYTDANKIRLTAMLRGTSV
APAKAALIEFVMKPVDVNLDPVSGNVIYSNPVIALGDVSKPFKPREDDYLFFPDLKLAANMDGSVSDEITQRDGKYWKRS
CFTEKAIDPKDFGTVNVFNLSGFKEVDIGGFKDTGIRPLNAFGVRYDGTLLKYSPGASTGANYFDFNETATLYITIPNAD
SGWGDSYTPTADEVKAYFLGYKMYLAGGPGNVDYNGTGTKAWAYRTSAGYQEAGITLPTTQAPNYTPYRIAFQLAQPAET
EIIPEGSITLHEGLNHIETGVGLFVREHMTSASSGNYYTSNDLGNSSTLFKNRVRKVWSIYRNRRQDKQWSFNNLSSYGL
EKPVIEAQKFDPTAVYEVTYLALDPISAPLSSITASTDTNLKKVVDTLAQTQADVETRLSMLERSSPNKAQAQWITATLL
NGWVVNVVSPAYMRDGFGFVHLKGSTKSGAVAAGTVLFVLPPEYRAKSYGQYTLKSDNGTNAVYGTLAISEDGKVTIYHN
IGNAGLFLDGISFPTF
>lcl|NZ_CP026520.1_prot_WP_053228768.1_4260 [locus_tag=PC41400_RS21495] [protein=phage tail family protein] [protein_id=WP_053228768.1] [location=complement(4769940..4770737)] [gbkey=CDS]
MIDATVNGVKFSDIGLGLIRHNIPVLPPTNDYTLEIAGRDGEINFGSSYQARTFELECVVMANDPTLDYHAKIANIAGLF
NAKRGELEITFSDLPGRRYMARYAGSMEIEKLLFDGNVTIPLKMDDPYVQSLYDTKAKEYGQGLEYGQLYEYSDYSVQIS
SSSQSFTIRNTGNVDAAPTIRISGAFTNLSLSDGHNVLTITDTCTASDVYEIDGHHYTVKKNGINAYAKSNGVFFTIPPG
ATTFISTATNPDFKVEVIFRHKYLY
>lcl|NZ_CP026520.1_prot_WP_053228767.1_4261 [locus_tag=PC41400_RS21500] [protein=phage tail tape measure protein] [protein_id=WP_053228767.1] [location=complement(4770734..4775338)] [gbkey=CDS]
MAMNEKDVVGARLKLDTSKLLPAFQVIDGGAKKNAESFRLLNNEIAQTERVYGDLAKAMDKSALSAEERRKKILDESRAL
VEQRTAQAELLRAKKESLEATNQVVEAKLATQEAIVKKRNQAIEQQEREHLQRMQTLQSKSQSASSQEALAQAKLDQKLQ
QIQQSNAREEREAERHAQRMNQISQQTGQMVAGPSIASNLGERLQQAALHATVYQGIYAAIHATQQAIHDGLVKIESNMA
GYVQTNEHYFVSFNKSTGEMVMNTQRLNEETTKFIQTAHDLGADINDVTESARLWGRMYKDVNVVQEMVRQSTKLSTVDM
VALSDSTKMMEAVMAQYAVHIGNANDAMVIGNRVLDSWSKVAHDTMAPARDLGAAFERTGKIASETGVSFDFMNGLISAG
IRNTALSGENLGNMWKTVLGTIRTDKAVKELEELDVRTKEVVNGVEQWRKAEDILLDLSVKVIDKNYDLTKSYADISRGV
YQFAKLAASLNVGDILLGQSASINSTGSTMEYLKVQMDTIERKAKQTKTSLLAIFNQAGEDGLRTTIKNALDVLDQLLIG
LTKVPKEVYQGTAALAGLVVAYKAIYQPIMSWVAAERALSAAKSISIMATNAETTAIYAQTAAMAQAARMTALATGGLTL
LVGVISLYVFNLGAAEKAERNQREEMEKKIAVQEQMINQTERQSEFLTKAIDVRNQLTQALKNQSLTEEQKNNITNQLSK
TEQAITAVIGEEGLQRLKTAGYTKEAIQAEKEAIESKRQVMIAASNEMIRKDQQETQTRIAEAKKRMAAMGEEMKVLQAR
ANMQNSSGVSSIKKSVFGFFGLSVPETYTERLLGEKQNEYNKLQDTINEGNARLNELSGKLVDNASKSLDPLSGMSGKGE
RNDSTEDQKKSFQAEMNKYRHLVNMKKEGYVSAADQAKQLKAIRDKYKGSLSEEDLYGIDEDIFRAGEGVSTKAKGFSAG
KQPKAPKAPKQREPYKFPLDAIDKDIKEAKMVVDSADSVIDFYNAKRGAMSGVIDDYAQRIELFDNRQKQLHVSNESLRA
SLGELQGKQAELNRLYKNGTITLDEYNRFSEDVTTRIASLTKEIDANSIAWWNDAKAIKEVKEQQIQDSFEFSEKWISHQ
KAIGELAAEGELAAWKRVQARYLEGTEQRKRADEQVYAAKKALMQEEESRLKDKATKEEKSVEESFKTQKKLLNDLKKAE
LDAIKTARDQYVDAQDAKIKALEKLIQLEDESNEDDDYEAELAKKKARAKLLEDAVGPEGKKERRDLLVEIEKMEEEHRR
KLRKRGLESLKEQYQEEKTAKEKEFDEKKQAVETHYDSLLNVFDQFKNDVEGRAEALKQLQILKESEKNAEILKQLDAFI
DSYKSKMTTIAGYAPADGGETERERDLRTYNSNIDKWYAGDDAEKARVHDENQHLRWKYGIEEDTGKLQHFSEGGIVRGK
PGDAVRVVAHADELILNPAQQGNLFRLLNMGMPKIQYDTSTYSKPAEIIQKITYNYEINSGDVNLSDEADIKSYWRERDV
MVRRFQSRGGVKTV
>lcl|NZ_CP026520.1_prot_WP_042232948.1_4262 [locus_tag=PC41400_RS21505] [protein=hypothetical protein] [protein_id=WP_042232948.1] [location=complement(4775638..4775967)] [gbkey=CDS]
MPELNMNESMNIGPEIKLYEGCTKHLKIGTIKLIRQVRSMTKEIRYQFMYCIGRGVVEKDGVKTDFDAEEARYKEIFQLL
VVEGLTDDEYEQIDENGLAELDGLLSRFL
>lcl|NZ_CP026520.1_prot_WP_042232946.1_4263 [locus_tag=PC41400_RS21510] [protein=hypothetical protein] [protein_id=WP_042232946.1] [location=complement(4776024..4776809)] [gbkey=CDS]
MQPLVFDGVGTIQTYGLDGKLKYVEGKISKVTMQFQTDWDRVMGGDSGYAFHYTAKDLQDKVSFEVPRYSTILAEISQGA
DTITGEVKFDENEEGILDSTNGYTVKAPTKFGGTFVPGSDAVYLKDANGKLTELTRAASAPTALQYAIDATGKITSDAAN
DNKNIMVTYKWSKTDGTRSGFSGKRRPVPIKFVHRFELTDDKTGKSVPCQLTIYKALGGGTLDASQERKKPTTNTVTLEI
MDPDITPDNPEGYAAELIFGI
>lcl|NZ_CP026520.1_prot_WP_156972856.1_4264 [locus_tag=PC41400_RS29930] [protein=hypothetical protein] [protein_id=WP_156972856.1] [location=complement(4776822..4776992)] [gbkey=CDS]
MARAGQKETEWDILVAEKQRLWNEVKALPDFVQSEQYRRIEEIDADLWRIKNSTKN
>lcl|NZ_CP026520.1_prot_WP_042232944.1_4265 [locus_tag=PC41400_RS21515] [protein=hypothetical protein] [protein_id=WP_042232944.1] [location=complement(4777006..4777440)] [gbkey=CDS]
MSVELTDAITGTLKSDDELKGLLGVMPNATPAQWLPKLTVGMEPEITISKDTVPHLCVFDKPGRFSMNPLVYEGKFCVDI
YAKTAAQARQIGSRLFHLLHDRQIMKSGFRSFLCVLAYEIDFATGITGVKGYQVIYDVDYIRMN
>lcl|NZ_CP026520.1_prot_WP_042232942.1_4266 [locus_tag=PC41400_RS21520] [protein=hypothetical protein] [protein_id=WP_042232942.1] [location=complement(4777445..4778026)] [gbkey=CDS]
MTDGYKGFNAEKASRDMETALSVQIAGLCGLVLQTAKSNVRYYPEVRNHLERQIFVLAHEMIAGEVTVDYWQAWLEQFGK
GSKMAGSSENPGLTSYMNSDLWNRLRPSGSRVVVGRKKGNYRSIDGTVRLSGGSYAGVDLEELAARGDIDSSYGPTPPSY
FLRAALQANRNRILQGLQEVIEGFPYHKYFEMG
>lcl|NZ_CP026520.1_prot_WP_042232938.1_4267 [locus_tag=PC41400_RS21525] [protein=hypothetical protein] [protein_id=WP_042232938.1] [location=complement(4778023..4778595)] [gbkey=CDS]
MFYDFSHRHASCLLDGQPSKVILSRENRGGSLASKELAYNGVFAPDSPVKTGSHISADEAYFVLSVRPTAEADKYCSLLK
INTKVEVQRWAALFDAEERHMGSDFLPVVADVPAVAQFVTARLRQDDAGLLPTTAYVLYVQNSVDVRRPQDKDLTSPDRV
LLNGRGYQVDAVDDIKVPGLLHVQLSEDVR
>lcl|NZ_CP026520.1_prot_WP_042232935.1_4268 [locus_tag=PC41400_RS21530] [protein=hypothetical protein] [protein_id=WP_042232935.1] [location=complement(4778604..4779194)] [gbkey=CDS]
MSAYLTLADTDYVSPSVTLNQSLIVQASSIIDGLCRRSLEVTSYTERIPLVDSLPGGSGYQRGHLTYFPVVAIISAKGRP
QYNALSGNFFGPPGFEDIADLSVIDVDKQTGSVACGFNPFGSPYSELEVTYTSGWEVIPDAIKAACGLFINQLAANPNAN
VKSKKDFDFSIEYFGNSYVTPDIADLLRPFILHSFR
>lcl|NZ_CP026520.1_prot_WP_042232933.1_4269 [locus_tag=PC41400_RS21535] [protein=hypothetical protein] [protein_id=WP_042232933.1] [location=complement(4779191..4779502)] [gbkey=CDS]
MTKAKNAKEQEEGIEPKQDDGVALQGVDSSQNLDQEKAGPDPAANEPDNKQPSNETVQLIGTDSERTGSHTLFYEGAALN
FINGEIEVKAELAAKLREAGFVE
>lcl|NZ_CP026520.1_prot_WP_042232930.1_4270 [locus_tag=PC41400_RS21540] [protein=hypothetical protein] [protein_id=WP_042232930.1] [location=complement(4779573..4780571)] [gbkey=CDS]
MNRVGIARAVDISAAMQFQGPGTLITDDFQREITDALRRNSILDGRLNYTPATGDISTYYEQNTINGGQAVDPRNPTSTA
TSNPRTPHGLKVKAITNEVNFGHYDLKLGQQQNNFAELKAKDLNDMLNGIGLTHARMLWRGTDTNLVLPNTLEYVGLAKQ
ITNTFTVSVTDSIITAIRAKVAAMIASEKYELMPTAIYMNPIAHHYLEVEVEQSGNSQTQISQLSKTTVAGIEVIAIMTA
AGILPIIPEPFMPAEVNATDNKLTDYGIAIVTEPMIEYHYVGEKNMYLFQMGTTTNLREDYVGIKYGAPVAKGPSYAHAY
GKVTRPTIAAVG
>lcl|NZ_CP026520.1_prot_WP_241688715.1_4271 [locus_tag=PC41400_RS21545] [protein=3-oxoacyl-ACP reductase] [protein_id=WP_241688715.1] [location=complement(4780573..4781601)] [gbkey=CDS]
MKTLLLKMSSAIKPQGFYLEASDKLEHPNKLPFKCALFAVDKPSDGPPGGAGGKRIRISSTVCEKHIHTFVGMALNIDYE
NGMQDHNSRFKVAVIEKAYLSMDGFAMVEGYIYAKDFPDVVGTIRYYNALAAETGYTEYQFGASLEMEASVVPAADDPEG
LLDVVDYCGTGAAILFAESAAYKTTRFAAKHKPKEEQKVNEEELKAMKEMLKNVAAGISTLTASVQSVASEVGTIKTEVD
QIKAANAEEKTKTGLQLAQERAEAAERELADLKAAAKDKPEDGARKTLSASQLLSKYGGKYDQDEMDASTFCAAVDKLGL
PTHESLKLKLQARRDLSEKETN
>lcl|NZ_CP026520.1_prot_WP_042232925.1_4272 [locus_tag=PC41400_RS21550] [protein=hypothetical protein] [protein_id=WP_042232925.1] [location=complement(4781688..4781882)] [gbkey=CDS]
MRRFRVTVVPDVVKKLGRQQAENDAEVYLKENIREAGEDENKYAIKCHIDVRTQQLICNATKVN
>lcl|NZ_CP026520.1_prot_WP_053228766.1_4273 [locus_tag=PC41400_RS21555] [protein=phage portal protein] [protein_id=WP_053228766.1] [location=complement(4781921..4783225)] [gbkey=CDS]
MGVWSWVTNFLQGGRVKNEPERQTDSFPFPFGFLGGRGTKQSTPKRTPANLRLLSESPIPRRAINIIKDGITKLNWSVTA
VNEEDQEKYQPLCTLIENVLKKPNSSDSFRLWLEQIVEDMLVASAGVSEKAKAGDVKHPLRLYPVDAFSIELYSNWDGKP
NSYRYAQRSDGKQVDLKDSELMYIRMNPRSSTPFGLSPLETVWESVTNFITVHRTSSTKANTNEQIINLGKGATPDDVKA
FRAFWEYEVKGRGQTPIISGESPSVLNLGTADDKALHIEWQRFLIEIVATAFGVSPKKLGQTKDVNRNTADSENDDTNDT
IFSIAENISEHINNEIIDGFFKMGDVLEFKFLYATSLKELKQKAEIDAMYLDRGVDTPDEVRNDRGKKALPNKHGEIVIR
RGYNVHGINKPLDDPRPGPEVPEDTGKEDKGVPD
>lcl|NZ_CP026520.1_prot_WP_042232922.1_4274 [locus_tag=PC41400_RS21560] [protein=phage terminase large subunit] [protein_id=WP_042232922.1] [location=complement(4783240..4784637)] [gbkey=CDS]
MSSKPFNVAVDLIDLYWDDPPAFAEDMLDFHPDPWQRAVMMDLAQNSHVSVRSGQGVGKTGLEAAITLWYLSCRPYPKVV
CTAPTKQQLHDVLWAEIAKWLNTSKMNNFLKWTATKIYMVGQKERWFATARTATRPENMQGFHEDYMLFLVDEASGVADP
ILEAILGTLSGLENKLLMCGNPTRTSGIFYDSHNRDRADYKTHKVSCLDSPRTSKENIAKLKRKYGEGSDVYRVRVEGEF
PRGESDSFIALESAEHAKSVRCQPSGTKLYVGVDVARFGDDETCIYGRIGGKVLKFHHHHKQDTMVTTGWVIRLINELRA
EHPVISEVEIRIDDSGVGGAVTDRLNEINSEKQLGYTIIPVNNGSSSDDESYANKGAETWGTVKDLLEQNMTNFVLGLPG
VLEVPDDEKLIAQLTTRKWRMNSSGKILLERKEEMKKRGLPSPDRADAFVLAFADIQVKSGFAFG
>lcl|NZ_CP026520.1_prot_WP_042232920.1_4275 [gene=terS] [locus_tag=PC41400_RS21565] [protein=phage terminase small subunit] [protein_id=WP_042232920.1] [location=complement(4784630..4785418)] [gbkey=CDS]
MSRERSPARQSALKMWLDSGRELKPKEIAAKLGVSDAMVRKWKSIDKWEEQPDPKPRRRGAPGGNKNAVGNRGGPGGPPG
NSKAVTHGLFRKILPDDPSYLEIFDATEDMTPLDMLWHQIRIAWTNIMDAQRKMFVKSKDEMIKELKKQKFEVHSTGRGK
NKELHQIVTEEEFEFQFAWDRQATALNAQTAAMARLASMIRQYEDMLRSLPPEDVLEEQRLRVTKLKAEVDMLNKPGEGA
EMWTSAIEEVAARRKGMRKVHE
>lcl|NZ_CP026520.1_prot_WP_042232918.1_4276 [locus_tag=PC41400_RS21570] [protein=tyrosine-type recombinase/integrase] [protein_id=WP_042232918.1] [location=complement(4785665..4786252)] [gbkey=CDS]
MKSVQPIRDKAIIEAIKRDLRLQSERNYIFFCLGIYSGLRVSDLLNLRVGDVRSKSHIVLIEQKTRNTKKLRKERKFIIH
PDLKPALESYIAGKRDEEYLFPSRQRKKKTGISQQPFHRTMAYKVLKNAAQKYGLKEIGCHTMRKTWGYHLYVQNPRNLA
LLMNMFNHSEQSITLRYLGLTQDAMDEAISDLSYS
>lcl|NZ_CP026520.1_prot_WP_164977148.1_4277 [locus_tag=PC41400_RS29935] [protein=hypothetical protein] [protein_id=WP_164977148.1] [location=complement(4786227..4786379)] [gbkey=CDS]
MDGIRVKDPPQQPLVCKGCIWGRWEAASQFCSKQVCIRGGAVYEVSSTNP
>lcl|NZ_CP026520.1_prot_WP_042232916.1_4278 [locus_tag=PC41400_RS21575] [protein=hypothetical protein] [protein_id=WP_042232916.1] [location=complement(4786713..4787090)] [gbkey=CDS]
MSVMYMMEELFPKATKADIARAKEDLRKYREDKQKVLLFESEPPETEIQIRRQAALIKSTRRIERAISQITFADVRSVME
YRFLKGNSRAATILYFSGWHCCEKTIDRKITEGILSVADTLLYLD
>lcl|NZ_CP026520.1_prot_WP_156972855.1_4279 [locus_tag=PC41400_RS29940] [protein=hypothetical protein] [protein_id=WP_156972855.1] [location=complement(4787636..4787797)] [gbkey=CDS]
MRKIIFNWYLALGNTEREVVEFKDDTTEEEIEKAFDDWVWDRIGSDFWREEEK
>lcl|NZ_CP026520.1_prot_WP_042232914.1_4280 [locus_tag=PC41400_RS21580] [protein=hypothetical protein] [protein_id=WP_042232914.1] [location=complement(4787830..4788132)] [gbkey=CDS]
MAVSAIAFKARNKEGRYLCDGVECGDWEDEYLDTLILSDATLIIRNDLSKPDNDDIEDFYKFIASLPLSNDVTFIKEHYD
PVEVTLSEEDLKIVRERNDW
>lcl|NZ_CP026520.1_prot_WP_164977149.1_4281 [locus_tag=PC41400_RS29945] [protein=hypothetical protein] [protein_id=WP_164977149.1] [location=complement(4788165..4788341)] [gbkey=CDS]
MEIGTKVIMVNCYEARKYPDKVWVTRSNPWELGHGAKVVLLEGKSGGFSVDCLKIVEE
>lcl|NZ_CP026520.1_prot_WP_042232912.1_4282 [locus_tag=PC41400_RS21585] [protein=hypothetical protein] [protein_id=WP_042232912.1] [location=complement(4788369..4788605)] [gbkey=CDS]
MARLKFEMWKDGDGNIMSRFTDGKGRSTDSYWCGPPESIDHVGPEYLPQRHRHPNVRGGRHIEFIKRQYKIEVAKVRV
>lcl|NZ_CP026520.1_prot_WP_053228765.1_4283 [locus_tag=PC41400_RS21590] [protein=hypothetical protein] [protein_id=WP_053228765.1] [location=complement(4788855..4789157)] [gbkey=CDS]
MGPGRPAKECENEELVFEFENACRGGDYEFAPEYREELLQRLTATPRPIDEWHEDDGTVLWWSFPIEEPPYCGSPLDADW
PGYHTHWTSLIVPDNPKGQT
>lcl|NZ_CP026520.1_prot_WP_042232907.1_4284 [locus_tag=PC41400_RS21595] [protein=hypothetical protein] [protein_id=WP_042232907.1] [location=complement(4789183..4789803)] [gbkey=CDS]
MKKATKRFRKKYLSRAAKSITTGKISRFHILRAKNEINLMFDRGYLNVFRPWWYDQSDRWNELDFRVEYKKHALATAAEI
ENKTRINLKKLQEDYDRLPKHPPRIRKYREPKPQPIRKLKNPEEFKIIVLENGVKKVLSVIGEKVFVVRGYDFFIRHDGT
FWVVSDVKTGAAVSKSVGYKDAVAVAKKRIEENFDQYLKILEKFAG
>lcl|NZ_CP026520.1_prot_WP_042232905.1_4285 [locus_tag=PC41400_RS21600] [protein=hypothetical protein] [protein_id=WP_042232905.1] [location=complement(4789837..4790091)] [gbkey=CDS]
MGRGAGAGHENYGNNRMYWIASTNVNGTIYCAKVICVSARSALVQGSGELEKMLKEDKIKLPDDWTVRVVSEKSEHAFSK
LDSI
>lcl|NZ_CP026520.1_prot_WP_042232903.1_4286 [locus_tag=PC41400_RS21605] [protein=hypothetical protein] [protein_id=WP_042232903.1] [location=complement(4790125..4790340)] [gbkey=CDS]
MKKEEISELMYRLYIACDQAPYDTDVKELIQSAPIKMQKEFISRMIQEKLWDIHPDEEDLEAARKLTGYDG
>lcl|NZ_CP026520.1_prot_WP_042232901.1_4287 [locus_tag=PC41400_RS21610] [protein=hypothetical protein] [protein_id=WP_042232901.1] [location=complement(4790369..4790587)] [gbkey=CDS]
MKAIQGFQIEDIDGCEFVQSTNIGKLMISHHQDFDGYYTGMIHVIYPDASHEAITREESQSMLDSGEWLVVR
>lcl|NZ_CP026520.1_prot_WP_129112520.1_4288 [locus_tag=PC41400_RS21615] [protein=hypothetical protein] [protein_id=WP_129112520.1] [location=complement(4790646..4791095)] [gbkey=CDS]
MSMLDRMRGSNPTFVTAPDIVGDAMGTLDMFLHWEPVIHSYGLPIAFVLQDGQEKLCMPWGKLDAVFIGGSTDYKLGPEI
RWLVREAKWRGKWVHMGRVNSLQRLSYARDIGCDSVDGTGYSRFPDANIPEALNFLKHEQITLNIDEAL
>lcl|NZ_CP026520.1_prot_WP_129112521.1_4289 [locus_tag=PC41400_RS21620] [protein=DNA-entry nuclease] [protein_id=WP_129112521.1] [location=complement(4791279..4791524)] [gbkey=CDS]
MIAEIQYDGLGRMKYHPDFPPKHGQPFSEEELCYLAKYYKIDGRLSVSLALDRPEGSIQKKYLSLKSNGMLERYSELEYY
S
>lcl|NZ_CP026520.1_prot_WP_042232899.1_4290 [locus_tag=PC41400_RS21625] [protein=hypothetical protein] [protein_id=WP_042232899.1] [location=complement(4791565..4791855)] [gbkey=CDS]
MSRTSFNSSREAYAYILEEHEKAKELLERTKFDISYLWMNITKGNFEDDSKQEHALKELGYLAKSSYSLADSAARLSNLA
KALSAYFADEINGSVD
>lcl|NZ_CP026520.1_prot_WP_129112522.1_4291 [locus_tag=PC41400_RS21630] [protein=hypothetical protein] [protein_id=WP_129112522.1] [location=complement(4792350..4792580)] [gbkey=CDS]
MMIKAGTFWVWTEQAETARPEWFAGHPVWSDYLHAAPAEWVQQGFVREATTEGQIEISEFLDLDGLKNKNAPAYLG
>lcl|NZ_CP026520.1_prot_WP_042232895.1_4292 [locus_tag=PC41400_RS21635] [protein=HNH endonuclease signature motif containing protein] [protein_id=WP_042232895.1] [location=complement(4792570..4793166)] [gbkey=CDS]
MFKKKRKEVAPWRQGITAHHNRRPSKADRAEFPTTVIKELIQEADGRCQCCGLLPDTTTHHVMPRGRGGRGVKTNGMRLN
GVCHDRIQTNEEELQYWISEWTRRYGPYFWYDAQDFEELEKRKAKEFQMHEQAERLQPVMELLSNATGRKLKPSERRLVH
EIGAKPDRLMVFQQLLQDTVGPGITNMDKQFGYGRFDD
>lcl|NZ_CP026520.1_prot_WP_042232893.1_4293 [locus_tag=PC41400_RS21640] [protein=hypothetical protein] [protein_id=WP_042232893.1] [location=complement(4793147..4793461)] [gbkey=CDS]
MTAAAQSPYARMGNTNKRGKAVIGGKKHDRRRIENKVPVQLGEFVELIQEQVIDLDYTIRMLFVELIAQGFADHEVIMHS
QAILLKAIQKASSAEVMLNVQKKA
>lcl|NZ_CP026520.1_prot_WP_042232891.1_4294 [locus_tag=PC41400_RS21645] [protein=ATP-binding protein] [protein_id=WP_042232891.1] [location=complement(4793566..4794384)] [gbkey=CDS]
MRETLRPEELIAKIQHLKAIKPDGYGQAETPIYKCAKCKDELGYFVQRPIVMSQKGEAKEFMIDYWKDCDCIVDRKVDKA
FESSKISFEFQKKSFDNFDLEVLGPVRDAYQCTRTYTEQYRKVKLEKHNSIALLGASGCGKTHLLMAAANTLISQGVKVQ
YFPWVEGFNELKDDMALVKERVFQLQKAELLFIDDMFKPRKGPTDFELTQAFAIINYRYLEKLPIMISSERDIKTMCGYD
TALGSRIYDMCKDYLVNIVGDSYKLNYRLRED
>lcl|NZ_CP026520.1_prot_WP_053228763.1_4295 [locus_tag=PC41400_RS21650] [protein=replication protein] [protein_id=WP_053228763.1] [location=complement(4794275..4795294)] [gbkey=CDS]
MANPQVENGFVSVANELWDEIIRRNFTKRQKDILYFVLRLSYGCRNKSAFVPKLKDFELCGVGKTHITNELKYLVECRVI
AWDKAECAFAINKDYEKWQVSPVAGWNREEFNNLIHQNLAKKLPKQEHEIPEKVTEIVTSENNGYRNSNQQVTETVTEEL
PKQEPLSGLNPCGSKAEGVSKDSIKDSIKNNITTTDIEPAESDPLIQIQDAYCDLHEKLPFNLSQKDLSLMREVVELEIP
AQFVIKTMQDTFALKKRRATENVVPFKKPSSFTYYENPILDAWRNHQDKIKPFPKGGQSNVQVNAGDASTRRTDSQNPAP
ESNKTGWIRTSRNADLQVR
>lcl|NZ_CP026520.1_prot_WP_042232889.1_4296 [locus_tag=PC41400_RS21655] [protein=hypothetical protein] [protein_id=WP_042232889.1] [location=complement(4795791..4796057)] [gbkey=CDS]
MSNVNLKLARKRVGELTLLYRKILWIREPDGIWDWFLKREIEELEKDISAAKENMTARPGDHVNLKNIVVVSIPTFLTPD
KGAAGYGG
>lcl|NZ_CP026520.1_prot_WP_053228762.1_4297 [locus_tag=PC41400_RS21660] [protein=helix-turn-helix transcriptional regulator] [protein_id=WP_053228762.1] [location=complement(4796072..4796578)] [gbkey=CDS]
MSIGDAFRSYRKENDMTQQEFSEIVPLDRTGLSKIEADKKKAPKDIMRIAAIALDDPRIALAAQAEVTGGACPPWLSKAD
LHKTTVHLKALEEIEEAFEAMKSTPITKRKDQLEMKDIHRIKNTIMECVEAITALTHYVAILCKEYTISWIGVWKEHRQE
LKEKKYLN
>lcl|NZ_CP026520.1_prot_WP_042232887.1_4298 [locus_tag=PC41400_RS21665] [protein=hypothetical protein] [protein_id=WP_042232887.1] [location=complement(4796758..4796961)] [gbkey=CDS]
MNNYFQAYPLYLKASDIMDALQVSKTKAYEILQRPDCPTVTLGKSKRVHRDKFFEYIVSLEGKTLPA
>lcl|NZ_CP026520.1_prot_WP_042232884.1_4299 [locus_tag=PC41400_RS21670] [protein=helix-turn-helix transcriptional regulator] [protein_id=WP_042232884.1] [location=complement(4796958..4797206)] [gbkey=CDS]
MTKETIAEKLLKLRSKKPRYEVAKEVGISLSALQMYENGQRIPRDDIKVRIANYYKVSVEAIFFDTKPHESCGSNEERRK
TS
>lcl|NZ_CP026520.1_prot_WP_042232882.1_4300 [locus_tag=PC41400_RS21675] [protein=helix-turn-helix domain-containing protein] [protein_id=WP_042232882.1] [location=4797336..4797713] [gbkey=CDS]
MSKFGLRLRELRKKAGLTQADLAGKFNISESSVGMYERGEREPSLELLNRFSEFFKVKVDYLTGRESSSIEDDKDFLEIL
NDPEVNVFFKDFASAPKERTKEMLKYWRYLKNEEKDRKPGDKQGE
>lcl|NZ_CP026520.1_prot_WP_042232880.1_4301 [locus_tag=PC41400_RS21680] [protein=ImmA/IrrE family metallo-endopeptidase] [protein_id=WP_042232880.1] [location=4797791..4798306] [gbkey=CDS]
MENSAYQKTYLEIWIENLYRSQKITIPQQLNVIEVASRLNVWVYWWNSGSESFERNGMMSIFLDRRLPQSQQWLDFLHEL
CHLLRHWGKQFDMPKEFRELQEWQASHFVMYASIPYYMYESLPLPDKDSEAAEALKSEFGVPIEMAYARIQQIRRRILQG
ICDQRVNLLYG
>lcl|NZ_CP026520.1_prot_WP_042232877.1_4302 [locus_tag=PC41400_RS21685] [protein=tyrosine-type recombinase/integrase] [protein_id=WP_042232877.1] [location=4798321..4799397] [gbkey=CDS]
MAYFRRRGDKWSFTLDVGRDQNTGKRKQKTVSGFSSEKEARKACSKMERDLEAGKRTDSMKFDDFIKLFFETIVINKVEQ
GTYNSQWIIVQKYIIPKLGHKKIDKITAVDLDIFYSDLLKDGVSRGYIKNIAQVISKTFKQASIWNYIIKNVAKESSSPS
YKPKEMSIWTEEQLTHFLQESREFERHALYVLAGTSGMRIGEILALDWDDIDFKKSTVSITKSLKHTKQKGLHVKRPKTE
NSKRVIHIPHSTLQVLIEHKNKQLPGVSIVFDNKGSYFYTNNVWRYFISDSAKVGMPRIRLHDLRHTHASILLSWGYNAK
VVAERLGDTVETVMNTYAHVLPNMQEEVAIKLNQLYAH
>lcl|NZ_CP026520.1_prot_WP_042232875.1_4303 [gene=dprA] [locus_tag=PC41400_RS21690] [protein=DNA-processing protein DprA] [protein_id=WP_042232875.1] [location=complement(4799488..4800573)] [gbkey=CDS]
MENRSVLIGLHGITGVGWKTIDLLVRSCSDLKDLLLWSAGDYARLGIPAARIQRLIDELTPFKIEASLEKYESLGIGILT
RFDPGYPRLLEETAQPPWVLYCRGRQEVLQRPTLGVVGTRTPTAYGRKAARELSMALSAAGFVVVSGLARGIDAEAHHGA
KDGAGGTIAVLGVPPDRIYPPEHTRLHEEIIASGGLVVSEYPLGTVPHPGLFPLRNRVIAGLSLGVIVVEAAERSGSLIT
ADLALEESRDVFAVPGPIHSPKSRGALSLIKQGAKMVTDPRDIYEEYMHLLSTLSGQMGGSVQPGTLPSPEQQLTEDEVI
LYRLLSSVPVPIDVLLEQTQFTFGHLHSVLLSLLLKNNSSS
>lcl|NZ_CP026520.1_prot_WP_009674484.1_4304 [gene=sucD] [locus_tag=PC41400_RS21695] [protein=succinate--CoA ligase subunit alpha] [protein_id=WP_009674484.1] [location=complement(4800696..4801628)] [gbkey=CDS]
MSILIDKNTKVITQNMTGKTGMFHTLGGLEYGTQMVAGAVPGKGGTQVDFTLENGSTVTLPVYNTVQEAKDATGATASVI
YVPPAFAADSIMEAVDADLDLVICITEGIPVLDMVKVARYMEGKRTRLIGPNCPGVITPGECKIGIMPGYIHTPGHVGVV
SRSGTLTYEAVHQLTTRGIGQSSAVGIGGDPVKGSEFIDILRLFNEDPDTYAVIMIGEIGGSAEEEAAEWIKANMTKPVV
GFIGGKTAPPGKRMGHAGAIISGGKGTAAEKVATLEACGIRVAATPSEMGSTLVSVLEEKGLLEKCITKK
>lcl|NZ_CP026520.1_prot_WP_042232873.1_4305 [gene=sucC] [locus_tag=PC41400_RS21700] [protein=ADP-forming succinate--CoA ligase subunit beta] [protein_id=WP_042232873.1] [location=complement(4801691..4802851)] [gbkey=CDS]
MNIHEYQGKEVLKQYGVVVPQGKVAFSVDEAVEAAKELGTQVVVVKAQIHAGGRGKAGGVKVAKNLDEVRTYAEQILGKV
LVTHQTGPEGKEVKRLLIEQGCDIKKEYYIGVVVDRGTGRVTMMASEEGGMDIEEVAANTPEKIFKEIVDPAIGLQAFQA
RKLAYAIKIPQELVNKAVKFMLALYTAFVEKDCSIAEINPLVVTGDGNVMALDAKLNFDSNALFRHKDILELRDLEEEDE
KEIEASKYDLSYIALDGNIGCMVNGAGLAMATMDIIKHYGGDPANFLDVGGGATKEKVTEAFKIILSDSKVKGIFVNIFG
GIMRCDIIAEGVVAAAKELGLDRPLVVRLEGTNVDLGKKILNESGLNIVSADSMADGAQKIVALVK
>lcl|NZ_CP026520.1_prot_WP_042232870.1_4306 [locus_tag=PC41400_RS21705] [protein=hypothetical protein] [protein_id=WP_042232870.1] [location=complement(4803163..4803423)] [gbkey=CDS]
MQDKPYFCPNCRSNRIKFSLMTSYSQPFVKDALTGVVTESSEANAITQSEPTIQCQVCSFVGNEMRFVKQAEREPRTVTQ
VSPTYV
>lcl|NZ_CP026520.1_prot_WP_042232867.1_4307 [locus_tag=PC41400_RS21710] [protein=YifB family Mg chelatase-like AAA ATPase] [protein_id=WP_042232867.1] [location=4803589..4805184] [gbkey=CDS]
MYGKVTGACLHGIDGICIEVEIDIANGLPQINVVGLPDTAVRESIDRVRSAIKNCGFQFPLERITVNLAPADLRKEGASF
DLAIAAGILLTSGQIELPGMGGCLLVGELALDGGLRPVPGVLSMVDAARRRGIRRVLLPLSNVDEARLISGTEVQGISSL
SELGTPCKRGLTADGASLSETESASESFAREVLSARPDVPAEVHGPLAASDDYADVIGQHHAKRAIMIAAAGMHNLLFIG
PPGTGKTMLARRIPGILPPLSDEEALEVTKIYSVSGKLGTITRFMRQRPYRSPHHTVSAAGLIGGGSIPKPGEVSLAHQG
VLFLDELPEFSRSVLEVLRQPLEDRKVTIGRARASFTFPSGFLLAASMNPCQCGFYGTETAEQSCICSPLKIRQYKARIS
GPLLDRIDLHVEVPRLDYTAMTDKRPSLSSEDMRRQVLAAHRRQQLRYEGSFMRFNAELGGKWLRKYCVLPPEAEDLLRQ
TFDRLRLSARAHDRILKIARTIADLETSDGITYGHIAEAIQYRNLDKSYSL
>lcl|NZ_CP026520.1_prot_WP_042232865.1_4308 [locus_tag=PC41400_RS21715] [protein=hypothetical protein] [protein_id=WP_042232865.1] [location=complement(4805234..4805506)] [gbkey=CDS]
MNGLKFIRSVRNCSPDRQDDREKRRPSLERALFLVRTAWNLCGPDETPHDDDQSQVVWHAGKGMQRIFVTYREFTLAVFL
RDFEMTAVSR
>lcl|NZ_CP026520.1_prot_WP_042232863.1_4309 [locus_tag=PC41400_RS21720] [protein=YraN family protein] [protein_id=WP_042232863.1] [location=complement(4805507..4805881)] [gbkey=CDS]
MAFSPLSRKDLGRYGEQRAEEHIAALGYKVVARNWRCRSGEIDLIAETEGQLVFIEVRTRRLTGRFGTAKESVDARKQLK
VRQIAQMYLYAAKQSDRRARFDVITIELTPEGEFNRLEHLTGAF
>lcl|NZ_CP026520.1_prot_WP_042232861.1_4310 [locus_tag=PC41400_RS21725] [protein=EscU/YscU/HrcU family type III secretion system export apparatus switch protein] [protein_id=WP_042232861.1] [location=complement(4805883..4806257)] [gbkey=CDS]
MNFSAKRGQTPSNSGPSKETADGEGAGTHAIRKAVALKYAPGEQKAPTVIAKGKGKVAEAIVEKARESGVPVQEDASLVE
VLSKLELDQEIPGELYQLVAEVLSYIYRSDNRMKEMRDSQRKGL
>lcl|NZ_CP026520.1_prot_WP_129112523.1_4311 [locus_tag=PC41400_RS21730] [protein=hypothetical protein] [protein_id=WP_129112523.1] [location=complement(4806254..4808362)] [gbkey=CDS]
MNISGLIRGLAGGMQAAEPKTLELKPGQVVKGVVLQVYENQEAQMSIDGVQVRAKLETPLAPGQTAWLQVQPESTGGQVV
LKPAAGPAVPATQEEIGQMLKQTGLKDTAVNRELVQMMKQQGIPLTKEAAAPLQEVMSRIPASVKPEQWVQAVGVAVSKG
LPLTADTVGAIHQALFGKPFGETLQNLAQTLSGLLQNGGLAEGTSGSGEKLLRQLAEVLRTMQDTAAEVGGKPTSNGTAS
PLDNGTGKPGLAAGTPTANSGGGTGTAGQADTRQPGGIRPDGSGGTAQAAGQGRQVLPGTSGLPQTAAGAPADGGVRLTP
SAAGMQADAAAELPQGAGRPAAGASAPGTQAQQAGGTPAGQTAAQAGGSALAAAPGTTAGGPAGLTPAAGRPAAGGEPLA
QASPGTAAASGARPDAPAAQPQPGLPASAPRSAADAHAAQAAERRGSAAEADAARASAAGEQRPAADSGWVRELLRAVGV
DHESRLLRQPGTLAHAGGAHLAAGEGQAAALKADSAETLKSLLLQLTAADDLPAPIRESAKQALQQITGQQLLLSPDRSG
VFTHVTLFVPLLGADGQPTAAVHIQSRKGRRGELDAENCRLLFDLQMKSMGHTLVDVGVMNRMVSLQVFNDEPIIGELLN
GSKEEIAAALDKLGYQFLSMKTAEYPKASGSFTGGESGSSLSAVMMDTIVPGEAYRGVDLRI
>lcl|NZ_CP026520.1_prot_WP_042232824.1_4312 [locus_tag=PC41400_RS21735] [protein=ribonuclease HII] [protein_id=WP_042232824.1] [location=complement(4808418..4809026)] [gbkey=CDS]
MLHEETGLWEKGYSRIAGVDEVGRGCLFGDVVAAAVIMPQGLILEGVDDSKKLSEKKREELYDVIMKEAIAVGIGTIDVK
TIDEINIKQASRLAMKLAVEQLDPAPDFLLVDAEQVDLDIRQMAIIHGDATSQSIAAASIIAKVTRDRMCREWDKLHPEY
GIAKHKGYATRQHREMLIEYGPTPLHRKLFIRNVMVMQQELF
>lcl|NZ_CP026520.1_prot_WP_042232822.1_4313 [gene=ylqF] [locus_tag=PC41400_RS21740] [protein=ribosome biogenesis GTPase YlqF] [protein_id=WP_042232822.1] [location=complement(4809188..4810060)] [gbkey=CDS]
MTIQWFPGHMTRARRQIEAKLKLIDVTIELLDARIPMSSRNPMVDDILKGKPRLVLLNKADLADPAITAEWVKFFTDRGL
KTLAIDANTGTGVKEILPKCKELMSAKIETQIRKGINPRAIRGLIVGIPNVGKSTLINQLAGRKIANTGDKPGVTKGQQW
IKFGTDMDLLDTPGILWPKFDDQNVGMKLAATGAIKEEILHLDDIAFHTIRYMSEYYGERLKERYGIEELPEDSENPNEV
IEVMEAIGRKRGAIISGGRVDLEKTSLLFLRELRSGKLGRVSLEVPEEYL
>lcl|NZ_CP026520.1_prot_WP_042232820.1_4314 [gene=lepB] [locus_tag=PC41400_RS21745] [protein=signal peptidase I] [protein_id=WP_042232820.1] [location=complement(4810097..4810732)] [gbkey=CDS]
MMERREQEVSAETENVAAPAKNPVKKEAWEWIKALLIAAVLVFLIRWLLFAPFIVEGPSMEPNFHTGERLIVNKIKYKFS
EPKRGEVIVLHAPEGIDYIKRVVALPGETIKVDGDKVYVNDKVIEEPYIKEAIDNAVKEGHAYNRRNFPEPGSEGTTEIK
VPDKSVFVMGDNRSNSKDSRFAEVGFVSYDKIVGRADLVFWPLNKAGFVHF
>lcl|NZ_CP026520.1_prot_WP_042232818.1_4315 [gene=rplS] [locus_tag=PC41400_RS21750] [protein=50S ribosomal protein L19] [protein_id=WP_042232818.1] [location=complement(4810815..4811156)] [gbkey=CDS]
MNLIQEITKEQLRSDIPSFRPGDTLKVFVKVIEGTRERVQLFEGVVIKRRGGGISETFTVRKISYGVGVERTLPLHSPKI
EKIEVARRGKVRRAKLYYLRGLRGKAARIKEIR
>lcl|NZ_CP026520.1_prot_WP_042232816.1_4316 [gene=trmD] [locus_tag=PC41400_RS21755] [protein=tRNA (guanosine(37)-N1)-methyltransferase TrmD] [protein_id=WP_042232816.1] [location=complement(4811372..4812160)] [gbkey=CDS]
MNIDVLTLFPAMFEGVFTSSILGKAREKGIVNLSTVNFRDFAGNKHNTVDDYPYGGGGGMVLKPDPIFAAVESLLDVPSS
GSETVGEPVSTGAKPRVILMCPQGETFTQRKAEELSREEHLIFICGHYEGYDERIREHLVTDELSIGDYVLTGGELPAMV
VIDSVVRLLPGVLGNEQSAVTDSFSTGLLEYPHYTRPVEFRGWTVPDVLISGHHANITEWRRLQSLYRTFKRRPELLENA
ELTDKEKRWLREKMAEEEGRQG
>lcl|NZ_CP026520.1_prot_WP_042232814.1_4317 [gene=rimM] [locus_tag=PC41400_RS21760] [protein=ribosome maturation factor RimM] [protein_id=WP_042232814.1] [location=complement(4812157..4812678)] [gbkey=CDS]
MSDKLYTVGTIVNTHGLRGEVKVVPHTDFAEQRFANGSKLVIEDAKGKLTPVTIKSTRLHKNMYIAQFKEFSHINEVEVF
KSSLLRVEEKYLESLDEDEYYYHEILGCRVLTEEGEELGTIEEILTPGANHVWVVGRPNKKQLLLPVIDDVVLDVNVEEK
IVTVHLLEGLLDL
>lcl|NZ_CP026520.1_prot_WP_042232812.1_4318 [locus_tag=PC41400_RS21765] [protein=KH domain-containing protein] [protein_id=WP_042232812.1] [location=complement(4812860..4813090)] [gbkey=CDS]
MKEVITVIAKALVDHPEDVRVNAVEGENSVIYELSVHPGDVGKVIGKQGRIAKALRTVVTSAAVKENKRVTVEIMS
>lcl|NZ_CP026520.1_prot_WP_042232810.1_4319 [gene=rpsP] [locus_tag=PC41400_RS21770] [protein=30S ribosomal protein S16] [protein_id=WP_042232810.1] [location=complement(4813118..4813390)] [gbkey=CDS]
MAVRIRLKRMGAHKAPFYRVVVSDSRSPRDGRFIEEIGYYNPVAQPAVVKIDEEKALKWLQDGAQASDTVRSLFSKAGLM
TKFHESKLKK
>lcl|NZ_CP026520.1_prot_WP_042232808.1_4320 [gene=ffh] [locus_tag=PC41400_RS21775] [protein=signal recognition particle protein] [protein_id=WP_042232808.1] [location=complement(4813442..4814803)] [gbkey=CDS]
MAFEGLTGRLQGVFDKLRGKGKLTEEDVSEALREVRLALLEADVNFKVVKDFIAKVKERAVGQDVMKSFTPGMVVIDIVN
KELTDLMGGTHTKLAKANKPPTVIMMVGLQGAGKTTTSGKLAKMLQKQNHKPLLVAGDIYRPAAIKQLQVLGEQLKVPVF
SMGDQVSPVEIAKNGLQFAKDNNNDYVIVDTAGRLHIDETLMDELKNVREAVGPHETLLVVDAMTGQDAVNVAESFHKQL
ELTGVVLTKLDGDTRGGAALSVKAVTGCPIKFAAMGEKMDALEPFYPERMASRILGMGDMLTLIEKAQAGIDADKAKEME
RKMRNAEFTFDDFLEQMEQVKKLGPLDQLLDMLPGAGKMKGMKDLKVDDKQMGRVEAIVKSMTKEEKRKPEIINANRRKR
IALGSGNSVQDVNRLLKQFEDMRKMMKQFSGMMNPKAMKNMKKLGKGFKFPFG
>lcl|NZ_CP026520.1_prot_WP_042232807.1_4321 [locus_tag=PC41400_RS21780] [protein=putative DNA-binding protein] [protein_id=WP_042232807.1] [location=complement(4814893..4815243)] [gbkey=CDS]
MTEDRILEKTNRINLLFDFYEPMLTDKQQAVLKLYFHDDYSLGEIAEQFKISRQAVYEHVKRAEQMLEEYESKLLLLTRH
ERRLKLLSEAEELLLQSPGDLADRAADYMRRIVHME
>lcl|NZ_CP026520.1_prot_WP_042232805.1_4322 [gene=ftsY] [locus_tag=PC41400_RS21785] [protein=signal recognition particle-docking protein FtsY] [protein_id=WP_042232805.1] [location=complement(4815453..4816442)] [gbkey=CDS]
MSFFKRLKESISSKTEAVTNKFKEGLTKTRNVFVDRVDDLFSRRKKIDEEFYEELEEILIGADVGVSTVMELIEQLRSEV
RKRKIEDPNDLQPVLSEKLTELLKGSEDSSIKLAPGGLSIILFVGVNGVGKTTTIGKMAHMYKSQGKKVILAAGDTFRAG
AIEQLETWGQRVGVDVIKQQAGSDPAAVMFDAVQAAKNRGADILLCDTAGRLQNKVNLMEELNKIYRVIRREVPDAPHEV
LLVLDATTGQNALQQAKMFGDKTGLTGLVLTKLDGTAKGGIVVAIRQELNIPVKFVGLGEKMDDLQPFDSEQFVHALFGK
WIAEEKAEA
>lcl|NZ_CP026520.1_prot_WP_042232803.1_4323 [gene=smc] [locus_tag=PC41400_RS21790] [protein=chromosome segregation protein SMC] [protein_id=WP_042232803.1] [location=complement(4816586..4820164)] [gbkey=CDS]
MFLKRIELSGFKSFADRTELEFVKGITAVVGPNGSGKSNISDGIRWVLGETSAKSLRGGKMEDVIFAGSDARRAVNYGEV
SLTLDNTSQSLPLDYNEVTVSRRVHRSGDSEYFINKQSCRLKDITELFMDTGIGKEAYSIIGQGRIEEILSNKSEERRSI
FEEASGIVKYKSRKKESEKKLQETEQNLLRIHDLVTELEDQINPLREQAEKAIRYKELKEELKHSEIAVYVHQIDQIYVS
WSETNEKLTALKKEQLSLSTFVSQHDAQLEKHRWETKRLEEELEKLQETLLQISEDFEKCEGHGEVLKERKKNFASNEQQ
LEQTLVFQETRFSEKETELAGHREKIVHIGAQLFEYQAKLSEEENRLQGVSGGLNSGVEDQLKSELLDKMNAMAQCRNEI
RYAEQQLESLARRLQKLEEEGQKRTALQSDIAARKKQLETKLEQTRKEVEKIRSAYLDLSESLKKKQTLLEESSGMLRKW
EQRIDALTSRRDTMKEMQNDYDGFMHGVKEVLKAKNRPDGLKGIHGAVAELVSVPAEIELAVETALGGALQNIVVATEAY
GREAIAFLKRRQMGRATFLPMDVIRGRSIGESERGALTSSNGFVGIGVDLIKFDAQYHNIFSSLLGTVIIAETLEDANRI
AAKAHYRYRVVTLEGDIVNPGGSMTGGSLQKKSSNLLGRQRQIEELDGEIEASETQLQQLKTRASQARKEIVDDNANLEE
LRGRGEQKRIEEQQIRADLAPLEAEVRRMEEQLTLERDDRSTLLEERGQQERSIEENRRQLEELQQEEEALQQAIRDAES
SRKANESAKEELQNQLTDLKVRVASASQEKQSLQDQQRRLQQDLAEAKQELEMNRRQLWQLRAEMEFHAEESVKQVELLN
TLKLRKQKCMEDIEFKRSERAGWVDKLQQAENETREQRTQLKQVEEQCLQHEVRVNRLDVELENLLKKLGEEYEMSYELA
KERYPVPEDITGAQNKVRDLKRSISSLGDVNLGAIEEYARVSERYDFLSVQKDDLVEAKTTLYTVIREMDQEMSKRFKTT
FDAIRSHFVVVFVKLFGGGRADLILSDPENLLETGIEIVAQPPGKKLQNLQLLSGGERALTALALLFSILRVKPVPFCVL
DEVEAALDEANVTRFAEYLREFSETTQFIVVTHRKGTMEEADVLYGVTMEEGGVSKLVSVRLEDEEAMISAS
>lcl|NZ_CP026520.1_prot_WP_042232802.1_4324 [gene=rnc] [locus_tag=PC41400_RS21795] [protein=ribonuclease III] [protein_id=WP_042232802.1] [location=complement(4820639..4821340)] [gbkey=CDS]
MNRDVNQLLADMEIRFQDLDLLRQAFTHSSYVNEHRMANYQDNERLEFLGDAVLELTVSEYLYFKYPHRTEGELTKLRAS
IVCEPSLVNFAEKLNFGSYVQLGKGEEMTGGRSRPALLADVFESFIGALYLDQGLEAVKLFLEQHVFPYVSAEGKLLVID
YKTQLQEHTQHHNMGALEYRIVEERGPAHEREFVAEVYMDAKQLGKGAGRSKKEAEQQAASQALTKLNVPMER
>lcl|NZ_CP026520.1_prot_WP_042232801.1_4325 [gene=fabF] [locus_tag=PC41400_RS21800] [protein=beta-ketoacyl-ACP synthase II] [protein_id=WP_042232801.1] [location=complement(4821362..4822564)] [gbkey=CDS]
MSSLGQEVNTFWGNLLAGKSGVSAIESFDVSEYPTRIAASVKDFNPEDYIDKREARKMDRFVQFAVAASINALKSSGLNI
QEDTDPERVGVYVGSGIGGLSTWEEQHKILLEKGPKRVSPFFIPMMIANMASGQISMITGAKGPNSTAVTACATGTHSIG
DSFKLIQRGDADVMICGGAEATISPTGVAGFCALRAMSTRNDEPEKASRPFDTGRDGFVMGEGAGVLVLESLEHAQKRGA
HIYAEVIGYGMSGDAHHMTEPDPDGAARCMIKAIKDAGIAPEDIDYINAHGTSTPVGDKSETSAIKKALGEHAYKVAVSS
TKSMTGHLLGAAGGVEAVICGLTLENGIIPPTINLDDQDPECDLDYVPNVPRNSDVQVAMSNSFGFGGHNATIILKKYQA
>lcl|NZ_CP026520.1_prot_WP_042232799.1_4326 [gene=acpP] [locus_tag=PC41400_RS21805] [protein=acyl carrier protein] [protein_id=WP_042232799.1] [location=complement(4822714..4822947)] [gbkey=CDS]
MSDVLDRVKKIIIDRLGVEESEVTLEASFKEDLGADSLDVVELVMELEDEFDLEISDEDAEKITTVGEVVNYIKSHT
>lcl|NZ_CP026520.1_prot_WP_042232798.1_4327 [gene=fabG] [locus_tag=PC41400_RS21810] [protein=3-oxoacyl-[acyl-carrier-protein] reductase] [protein_id=WP_042232798.1] [location=complement(4823051..4823791)] [gbkey=CDS]
MLSGKAALVTGASRGIGRAIALELAASGADVAVNYAGSEQAAEEVVKEIEAMGRRAVKLRGDVSNSQEAEDLVKQTIEAL
GSLDILVNNAGITRDNLIMRMKEEDFDIVIATNLKGVFNCTKAATRPMMKQRSGRIINISSVVGVLGNPGQANYVAAKAG
VIGLTKASAKELASRGITVNAVAPGFISTDMTDKLPEEQKASLLQQIPLGKLGKPEDIAKIVRFLASDEGSYMTGQTLHV
DGGMYM
>lcl|NZ_CP026520.1_prot_WP_042232797.1_4328 [gene=fabD] [locus_tag=PC41400_RS21815] [protein=ACP S-malonyltransferase] [protein_id=WP_042232797.1] [location=complement(4823830..4824756)] [gbkey=CDS]
MGKIAFVFPGQGAQSVGMGKDAYAAHEGAKRVFDEADKALGYSLSSLIFEGPEDELKITYHTQPALLTASVAYLEVFRQK
GIVPDFTAGHSLGEYSALVASGVLGFEDAVRTVRARGEFMEQAVPGGLGAMAAVLGAERGALDELCRSVSAETGAVELAN
INCPGQIVVSGTREGVQAVVDRAKEIDAKRAIPLEVSGPFHSSLMKSASEKLADVLAGLSLQEAAVPVVANVTARPVTSP
DEIKDLLVRQVYSPVLWEDTVAWLIEQGVDTFVEIGSGTVLAGLIKKVSRSVKVVSVNSLEAALQAEI
>lcl|NZ_CP026520.1_prot_WP_042232794.1_4329 [locus_tag=PC41400_RS21820] [protein=beta-ketoacyl-ACP synthase III] [protein_id=WP_042232794.1] [location=complement(4824783..4825769)] [gbkey=CDS]
MNGIPVGILGTGKYVPDRVLTNQELESMVETNDEWIVTRTGIRERRMAAPEQATSDLALEASRIALEKAGITADQLDLII
VATVTPDMAFPSASCILQEKLGAKNAAAFDLAAACAGFIYGLANASNFIANGIYKYALVIGADTLSRITDYTDRNTCILF
GDGAGAVVIGAVPEGRGFKSFELGADGSGAELLKIAGGGSRNPATPESIDQRLHYIYMAGSEVFKFAVRIMGSAAEEALR
KAGLHKNDIDLLIPHQANIRIIQSALNRLELTEDKCMINLDRYGNMSAASIPVALAEAAEQGRLKEGDTTVLVGFGGGLT
WGASVLVW
>lcl|NZ_CP026520.1_prot_WP_042232792.1_4330 [gene=plsX] [locus_tag=PC41400_RS21825] [protein=phosphate acyltransferase PlsX] [protein_id=WP_042232792.1] [location=complement(4825779..4826768)] [gbkey=CDS]
MRIAIDAMGGDHAPAIVVEGALAAAKEWADAELLLVGDSAKIEALLTERPSNLRVIHASEIIEAEDEPVRAVRRKKDASM
VVAGRLVKEGEADAMISAGNTGALMTTGLLVVGRVPGIERPALAPMIPTLDGRGVLALDLGANMDSSPENLLQYAIMGSI
YRSKVHGMDRPRVGLLNIGTEAMKGNELMKAAYPLLEQAPIHFVGNVESRDILHAEYDVIVCDGFAGNIMLKTLEGTALA
IFSALKTELTRTTLTKLAAAVLKPGLSAFKEMFDYTEHGAAPLLGLTGLVLKSHGSSDAKAIKNAVRQARTALQNDLVGA
IATEIKNGK
>lcl|NZ_CP026520.1_prot_WP_036739470.1_4331 [gene=fapR] [locus_tag=PC41400_RS21830] [protein=transcription factor FapR] [protein_id=WP_036739470.1] [location=complement(4826752..4827366)] [gbkey=CDS]
MVLVIERLPKRTRQQQLLQAIEDNPFLTDEELMRKFQVSIQTVRLDRLELGIPELRERIKHMAERSYDQVRSLPIDEVIG
EIIDLQLDKSGISMFELTEDHVFSRTKIARGHHIFAQANSLAVAVINDPIALTASADIRFIRSAMLGEKCIAKAYVKSLS
KGKAKVDVQTFIGDEVVFEGSFVIFHSKKEQLRTEGGQANADRN
>lcl|NZ_CP026520.1_prot_WP_009673368.1_4332 [gene=rpmF] [locus_tag=PC41400_RS21835] [protein=50S ribosomal protein L32] [protein_id=WP_009673368.1] [location=complement(4827568..4827741)] [gbkey=CDS]
MAVPFRKTSKTRRDKRRTHFKLEVPGMVKCEQCGELKLSHRVCKNCGTYKGREIVQA
>lcl|NZ_CP026520.1_prot_WP_042232788.1_4333 [locus_tag=PC41400_RS21840] [protein=DUF177 domain-containing protein] [protein_id=WP_042232788.1] [location=complement(4827775..4828290)] [gbkey=CDS]
MWIQLKDIMAKGQPVAIRESLNSETLLKGRKDILSAGPIVTDLEAKAASDVVEVTGRVSVELEFGCSRCLTPFKQTLNIP
FREIFTSKPEVEAQASEDEDEIVHLVCEDKVELNPYIEENIQLELPLAPVCREDCKGLCPECGTNRNERDCGCKNERIDP
RLAGLADFFNS
>lcl|NZ_CP026520.1_prot_WP_042232787.1_4334 [locus_tag=PC41400_RS21845] [protein=SepM family pheromone-processing serine protease] [protein_id=WP_042232787.1] [location=complement(4828350..4829534)] [gbkey=CDS]
MNDKNHENQRYNEEREQEESEREVGFENGDDARERPVIRRAHPQRGSRRFFTSLLVGFLCFVILFYVPLPFFIYMPGTAE
AVHPMVSVPKSNNTDEKGSLMLVTVRVANANVVTYLWSLANKYDELQLKKDVLRGMSEKDYSQQQQFVMLTSQSDAIQTA
YKKAGVPFHIKGEGVMVTGLSPGLPAEKVMQAGDKVTKADEKDIKTSQDLFAFLENKKEGDTVTLTFIRGDKEMTAPLTL
AKLPPDKDTGKSRVGIGITMAEMQSVQADDPSKQVTIKAGEIGGPSAGLMFSLEIYNQLLPEDITKGHKIAGTGTIDTEG
KVGVIGGIKHKVIAADREGAEIFFAPQDLKTDKGTIYNYTDALARAKEIGTKMKIVPVETMDDALKYLAELPPK
>lcl|NZ_CP026520.1_prot_WP_042232785.1_4335 [locus_tag=PC41400_RS21850] [protein=hypothetical protein] [protein_id=WP_042232785.1] [location=4829664..4830962] [gbkey=CDS]
MPKSLTHRGGGYRTTLLLGSLAAALVASIIAFPDKALQASLEGITIWWNIVFPALLPFMILTELLLGFGVVHALGTLLEP
LARLLLRLPGTGGLALAAGALGGFPSGALFTAKLRGRKLLTRGEGERLLALSHLASPVLIVTVIGTGFLHSPRLGLLLAG
AHYGGALLLGFVLPRQKKPAGAPAPVKRRGAGSGASGAPLWRQAAGSLARARSEDGRAFGQLLGEAVTASVQSLMVIGGY
IMIFSVIIHMLTMPDMLQAAGGLLGLPVGPDLLHHLASGLFELHLGAHAAAGEQALPQSIKLALIAAFLGFGGLSSHAQA
LSAAYGAGLRYLPFLMARLLHASFSYLLVLLLWRFTSGLFPSVSSGRQTLEVSGTLPPIAETGERFLFSSLNVWMPAFAQ
LGLLLLVLAAGCILSYALHAFAQRRLHAGPKA
>lcl|NZ_CP026520.1_prot_WP_042232783.1_4336 [gene=coaD] [locus_tag=PC41400_RS21855] [protein=pantetheine-phosphate adenylyltransferase] [protein_id=WP_042232783.1] [location=complement(4831012..4831524)] [gbkey=CDS]
MHQEKPRLKIAVYPGSFDPVTNGHLDIIRRAAKVFDKLIVAVLNNTSKNPLFSLEERIELLTKVTRDIPNVEVDGFRDLL
INYMESKDVHIIVRGLRAVSDFEYELQLSTMNNKLNEEVETFFMTSSPQYSFLSSSIVKEIARFHGPVSHFVPREVEEEL
KLKFAGTRQP
>lcl|NZ_CP026520.1_prot_WP_042232781.1_4337 [gene=rsmD] [locus_tag=PC41400_RS21860] [protein=16S rRNA (guanine(966)-N(2))-methyltransferase RsmD] [protein_id=WP_042232781.1] [location=complement(4831555..4832121)] [gbkey=CDS]
MRVISGSARGRSLKAVPGMSTRPTTDKVKEAIFSMIGPYFEGGEGLDLFAGTGGLGIEALSRGLDTCIFIDLEKKSLDTV
RHNLEVTGLTDRAEIYRNDAGRALKALEKRGKQFGLVFLDPPYRFKITGELMQKMKGLGLLEQGATIVVEHDAADKLPEV
FEGFVMQRRADYGDTAVTLYTFENTGLN
>lcl|NZ_CP026520.1_prot_WP_042232773.1_4338 [locus_tag=PC41400_RS21870] [protein=FxLYD domain-containing protein] [protein_id=WP_042232773.1] [location=4833354..4833818] [gbkey=CDS]
MKKILFVSVAALVLAAGCSSEQPAKTSSQTTTPTASTAPSETPKATPETTTKPTEAPKAKEGEYTAGDFKITGVSVKESA
GMWMVTAKVQNTGDKDIQAAILSAVLYDKDEKRLGIAQGGVEGMKKGESKTVQFVSIDKLDGYKKVEFQVDTSI
>lcl|NZ_CP026520.1_prot_WP_042232769.1_4339 [locus_tag=PC41400_RS21875] [protein=GNAT family N-acetyltransferase] [protein_id=WP_042232769.1] [location=4833974..4834441] [gbkey=CDS]
MSEGTIRTIDFNDTGLVRSLYELQRESYLIEAKLINFYEIPPLKETFEEFQACGEQFRGYFLDNQLAGAVSCTVEEETVT
ICRMVVHPDHFKKGIAQQLLNTLESDMGHVSLFKVATGKDNVPAKNLYAKNGYKHDRDLEVAPDFYISLFQKAVR
>lcl|NZ_CP026520.1_prot_WP_042232767.1_4340 [locus_tag=PC41400_RS21880] [protein=flavin reductase family protein] [protein_id=WP_042232767.1] [location=complement(4834558..4835115)] [gbkey=CDS]
MRTPTDDQAWYSYPGMVAVVTSRYGGRQNVMASGWHTYIGSSPGIYGISLRKETHTYSLIEKSGVFGVHFLPSSKSELIQ
AVGTFSGRDMDKYKELGIAYDNGLAADIPVLRDAYYAYECRVMDIRTYGDHEWIAGEILQRYKDEELFLDNGLPDLGKLQ
IPLYMGRSTYRIVEENTKEARHLFG
>lcl|NZ_CP026520.1_prot_WP_042232764.1_4341 [gene=lepB] [locus_tag=PC41400_RS21885] [protein=signal peptidase I] [protein_id=WP_042232764.1] [location=complement(4835182..4835736)] [gbkey=CDS]
MKIVREVLGWAGSIGISFALVLFIGIFVFQPYKVDGHSMDPTLQDEQRIYISKLTHTFAQEPAYGDIVVIDSRVDRPRTF
KDSFMEQPVIQWIAGSAKEKTFFVKRVIGKAGDVLEFKDHQVYRNGQVLSEPYLKETMQFVSDKKWTVPEGHVFVMGDNR
NDSWDSRNIGFIPLDHVMGVKKLP
>lcl|NZ_CP026520.1_prot_WP_042232762.1_4342 [locus_tag=PC41400_RS21890] [protein=cation:proton antiporter] [protein_id=WP_042232762.1] [location=4836021..4837181] [gbkey=CDS]
MLIFQLAIILLASKLAGDISVRLGQPSVLGKLLIGIVLGPNLLGLVTNTDVLQEISQIGVILLMFIAGLETDTEEFKRTG
KASGYVGVAGIIVPFGLGYLAGILLNQSPLEASFLGLLLSATSVSISVQALKEMGKLKSREGTTILGAAVIDDVLVIIAL
AFLMSMAGGDVNLGMIVVKKVVFFAAALLLAWKAVPWILKKFAPLRVTETVITAALIICFGYAYLAEYAGVAAIIGAYIA
GVAISLTNYKHEVFEKVETIGYSIFVPVFFTSIGVTVQFQGISSSIGLIVILSLLAIATKLAGSAIGARLAGFKWRSSLA
IGAAMVSRGEVALIIAAIGQESGLLSAEMFAVIVVVVLVTTIVTPPMMKLFFSEKSAPAAAHHSAE
>lcl|NZ_CP026520.1_prot_WP_042232760.1_4343 [locus_tag=PC41400_RS21895] [protein=hypothetical protein] [protein_id=WP_042232760.1] [location=complement(4837336..4837644)] [gbkey=CDS]
MSYLKLLQIYENDGLTPKRATSFRYDQMLGTILQEMKPTEVGSSMVLCRVDDFRSFQTTRVQAIDRDGSEFRVLTRNTIY
HFEIVESRNVSPASRRALCETS
>lcl|NZ_CP026520.1_prot_WP_042232758.1_4344 [locus_tag=PC41400_RS21900] [protein=HAMP domain-containing sensor histidine kinase] [protein_id=WP_042232758.1] [location=complement(4837696..4839051)] [gbkey=CDS]
MRTTINQHRKPVWIGSLLGLLTACILIGQAYAFLWTEQQKANSRNHLFWDSYLNAMYGVDKNWSSVIKQLKEDSDPQRLA
GEVGLRLYNADQKLLFDAAASSSSDKEDPAAGRIDLRTVKDGSRTLGYYSLPASPFHHPEPPFWAAVLAGSLLVGAAAGL
YLKRGDNRLRAVLDRILFRLAVLAREDGSEPSFRRGKENGGTDIHARTAEIDALLTGIEDKLRQMETVRPRMVADLTQEL
RRPLTVLRSSLEQAQIEEKRLTTDKAAVLFEEVSRMSALVNDMQQLMLAESGHLELERQWFSLRASLDPLIARLEAEARV
SGIELRYSPSREVSLYGDRHKLEQAFGHLLGNAVRLARSSASVSVTVNDLEVTVTVRDDGAGLAHDEIPHLFERFYRTAY
PSAAAASEPGLGFGMALVKQYVEAHNGSVRVTSEAGQGTSFTVRLPIFTGT
>lcl|NZ_CP026520.1_prot_WP_042232757.1_4345 [locus_tag=PC41400_RS21905] [protein=MarR family transcriptional regulator] [protein_id=WP_042232757.1] [location=4839249..4839713] [gbkey=CDS]
MLDNEDYVARILRATMSFYRRIGPQMSHFKEMGLTGPQFHVLNWISETGPGKITQLAEMMEVKPSAITVMIDRMEQNKFV
VRHHDENDRRVVMVSLTEHGQSVLKEAKLKSFEVIKQYYSRLDPDELESLMKISEKLQNIAGNEDDNNGKCCKS
>lcl|NZ_CP026520.1_prot_WP_174446837.1_4346 [locus_tag=PC41400_RS21910] [protein=MDR family MFS transporter] [protein_id=WP_174446837.1] [location=4839771..4841357] [gbkey=CDS]
MTKHRGMLIAGLIIAMLFGALEGTIVGTAMPRIVGELGGLGYMTWLTTAYMLTSTAVVPIAGKLADLLGRRVIYVTGLII
FMVSSALCGTTDSMTQLIVYRGIQGIGGGIMMPMAMIIIGDLFTGKERAKWQGVFGGLYGLASVIGPQIGGWIVDGLNWR
WVFYINLPVGIIAVIFIAMGLNKHKTQGPVKFDVAGMFTMIVGVVSLLLGLSFGGSEYPWLSWQIIGLFVLAVIAITAFI
VIESRTEEAILPVRLFKNRTFTVLNGIGFLMSIGMFGAIIFVPLFMQGVVGVSASASGTIMTPMMITMIIASIIGGQIVY
KIGPKVQISIGMIVMAAGFLLLTTMSIDTTKLTATLIMFVIGLGMGLVMPILTLVLQESFPKSELGVVTSSSQFFRSIGG
TFGMTILGAIMNHKSNSQLTDTLVPVLQKLPEQAKELVAKFTEMIHTNPQGLYTFLFSPEAMKEMPAEFLKIIAPILKNS
LVSSLHSVYLFALIFIVLGAVLTVAVPTIKLTERKKAEPAGKSEPAAE
>lcl|NZ_CP026520.1_prot_WP_042232751.1_4347 [gene=abc-f] [locus_tag=PC41400_RS21915] [protein=ABC-F type ribosomal protection protein] [protein_id=WP_042232751.1] [location=complement(4841537..4843447)] [gbkey=CDS]
MTLLVKAVGLMKEWNGRTLFENAGIEIYEGDRVALFGRNGTGKSTFFRLLTGDLQPDQGSVERRLPAAEWGWMRQQAEVD
PQTETLEYVQAYSNPDLHTLKTGIRELEARMAEEEPAGAKGGESAYEAYGELMERYARLGGYDWEVEVEKRLQQLKMGPE
CWRIPFLRLSGGQKTRAQLAALMVSSPKLLLLDEPTNHLDSESLEWLEQWMRAYPGTILFVSHDREFLDRTATSLYELTE
KGTVRYSGGYSAYKTQKELEMRTQEGLYKKQQRARRELEECIRNYKQWFQQAHRAATKVEVGVTKSFYLAKANKNRSRFK
AKERELERLEAEQVDKPRGGPRLNMELAGEDFAARTLLRAENLRLAFGEMPLFEGLNVTMERGDRLAVLGPNGAGKSTLL
KLLLGRIEPDSGEVVHHPRLRTGYFSQELEGLAGEETLLDTLLVLPSMTETAARTILGCFLFPADAAFKKLSDLSMGERC
RAAFLRLYFSGANLLVLDEPTNYLDIDTREQIEAALERFPGALAVVSHDRYLIRRTANRLLMLHPPDGAPEFFRGTYAEY
ADRDQSDASAPPSPRSDEIRRLELRLARLIAGPEEGGPEGAPDRMGAIRSLRDRIESLRTGGEEAPFPDSGTDPEG
>lcl|NZ_CP026520.1_prot_WP_042232748.1_4348 [locus_tag=PC41400_RS21920] [protein=hypothetical protein] [protein_id=WP_042232748.1] [location=complement(4843593..4843853)] [gbkey=CDS]
MSGGMDSSERKHVNNDDRPPIRRDCHAYVCAPIVRRFFSGCTSIRQPEAGYIARGMDASRTSGGPPGTVNSRLAGGTALQ
PLPQGK
>lcl|NZ_CP026520.1_prot_WP_129112524.1_4349 [locus_tag=PC41400_RS21925] [protein=hypothetical protein] [protein_id=WP_129112524.1] [location=complement(4844052..4844249)] [gbkey=CDS]
MNRSELARTSPNRVNTVRRGHCTVPVLGKSMTFGRRRTRYRTLDADLRQRGAQPAGPGKAMAAHG
>lcl|NZ_CP026520.1_prot_WP_042232746.1_4350 [locus_tag=PC41400_RS21930] [protein=SOS response-associated peptidase] [protein_id=WP_042232746.1] [location=complement(4844310..4845020)] [gbkey=CDS]
MCGRYTITVTVEELMARFGLFDAPGIPYHRPKYNVAPGQMVLAVVNDGRNNRIGELKWGLIPEWAKEESVGAKMLNARAE
TAADKPAYRIPLRRKRCLIPADGFYEWKREGGLKQPMRIRLKDGGLFAMAGLYDTWLSPDGRRVSTCTVLTTAPNPLVAD
IHDRMPVILRREDEAFWLDRQVQDPADLLSLLRAYPAAEMEAYPVSQLVGNVRNDSPQLIEPIVPPDSPVQEGWLL
>lcl|NZ_CP026520.1_prot_WP_042232743.1_4351 [locus_tag=PC41400_RS21935] [protein=HD domain-containing protein] [protein_id=WP_042232743.1] [location=complement(4845267..4846808)] [gbkey=CDS]
MRTYTYRRLLLALLRNYLIGSAIAVLGVGSVLIYSTLRATSREMSVMLVILGVSLLVMIVAELTVFARQMRPVRTVFLKH
SPALDEIKEAYIRLHRLPVLAVRRIFGPHLFGMVLPAVALAVGAIRMGMLEISYFDLTLAVLGALLVACMHAMIEFFLTA
YSIRPALRHSHDLALRLHGAEVSLDGKVLFSIRKKFQWSAFLIGAFPLFLYYLTVQLRLVREDTSLLKDYWNWDAAIILL
GILFASLGAWLLSRDIEHPIGQIHKVMGEVRAGNLAIYAPDLYPDEFSRLVAGFNHMVEGLRERDRINHELISGYFAALA
AALDARDAYTAGHSERVAGYSVRIGSLAGFSEHDLEELRKTALLHDIGKIGISDTVLLKEGRLTDDEFAQIQMHPALGER
ILRQIEPQSLMAPFLPGVRSHHERFDGRGYPDGLKGLDIPLFGRVIAIADAYDAMTSDRPYRKGMSAEKALSILKEGRGA
QWDPHFATLFIGWLEEGGPDKSAGAERTREHAG
>lcl|NZ_CP026520.1_prot_WP_042232741.1_4352 [locus_tag=PC41400_RS21940] [protein=hypothetical protein] [protein_id=WP_042232741.1] [location=4847019..4847423] [gbkey=CDS]
MQPPLQTRDDSLLVKRYVLLPVLLDVLEKNISLLESSGKLPEPLIGHLRLIQDRVTADMAFLRKQFRQRGIRIYEERRTT
EGLNVAYICRGYHHRFSLLWRLIRAEIVTALEVYLDPAAGSDTANTGQANITGC
>lcl|NZ_CP026520.1_prot_WP_042232739.1_4353 [locus_tag=PC41400_RS21945] [protein=DNA polymerase IV] [protein_id=WP_042232739.1] [location=4847522..4848769] [gbkey=CDS]
MTAFPEQGKRVIMLADCQSFYASVEKAAHPQYRDKPVVVAGDPARRSGIILAACPLAKAHGVTTAERLGEALRKCPEAVV
IRPRMEEYIKVSMQITAILQTYTDLVEPFSIDEQYLDVTGSLHLFGTPVEIARDIQQKVLQETGVWTRVGISENKVLAKM
ACDNFAKKNREGIYELPKSEIADLWELPVSKTYLIGSRMTRHLNRMGILTVGDLARTPPARLRARWGINGEVIWRVANGI
DPSPVTPQTHEGQKAVGHQMTLPRDYATAEEIKVVLLELSELVCQRCRAKGCQGQVVSVGCQGADYDHPSGFYRQMKLAD
PTQLTDTVYETAQLLFRKHWDGLPVRKVGVTLGDLVPDDQYQLLMFGDEERKIALERTTDTIRQRFGSTAIMRAVSATPA
GQAKDRSHKIGGHYK
>lcl|NZ_CP026520.1_prot_WP_042232737.1_4354 [locus_tag=PC41400_RS21950] [protein=YolD-like family protein] [protein_id=WP_042232737.1] [location=4848766..4849125] [gbkey=CDS]
MNSSSTGSKTGKKLQGNGLYESSRMMLPEHKEAYLRHQRELNRRAHPSLEDEERQRIGSLLAQAKTTGRVLTLTVYGPFE
DEEITGIVRRIDPLKQEIHLEVPGNVRRLAFDDILRISD
>lcl|NZ_CP026520.1_prot_WP_042232734.1_4355 [locus_tag=PC41400_RS21955] [protein=stage VI sporulation protein F] [protein_id=WP_042232734.1] [location=complement(4849341..4849628)] [gbkey=CDS]
MSYQQYGIDPALVERVKWKMKNPEIKERIKRLLEGVTKYDLQNRQKVAAFVAQFSKVLGEPLTQAQSSSLIDFVIAQKID
PNNTFHLLKLWGMFR
>lcl|NZ_CP026520.1_prot_WP_042232731.1_4356 [gene=recG] [locus_tag=PC41400_RS21960] [protein=ATP-dependent DNA helicase RecG] [protein_id=WP_042232731.1] [location=complement(4849957..4852002)] [gbkey=CDS]
MNLNEIPVTEVRGIGGQKAEELQALGITRVAHLLEYYPFRYEDYSIRDLTQVKDGEKITVQGKIMSEPVLQRYGRKNRLS
CKVVIDHFFVTAVWFNRPFLKDQLAPGREIVLTGKWDQRRHQMTVSDSEFPDRGTSKTGTLQPVYSVGGSLTQHFMRKAT
AQALTQYSAMVSEIIPAELLRKYRLLPRRRAMEIIHRPQGTEDGNEARRRMVYEELFLFQLKMHAFRAITRNRADGLAQQ
VDLPKVRAFVRGLPFTLTPSQKKVIAEILQDMQEPYTMNRLLQGDVGAGKTVVAAAALYATVTAGCQGALMVPTEILAEQ
HNRSLTAMFEPYGLQVALLTGSTTARQRRELLASLQMGMLDILVGTHALIQEDVFFRKLGLVVTDEQHRFGVNQRSILRR
KGMNPDVLTMTATPIPRTLAITAFGDLDVSTLRELPKGRKPIKTYWVRHEMFDRVLGFIRREADEGRQAYVICPLIEESE
KLDVQNAIDVHVQLQQAFPDLKVGLLHGRMTGAEKDEIMHRFKDGEIHTLVSTTVIEVGVDVPNATLMVVYDADRFGLSQ
LHQLRGRVGRGEHQSFCVLIADPKTEVGKERMQAMTDTTDGFEIARRDLELRGPGDFFGTKQSGLPDFRIADMMVDFETM
ELARDDAAELVARQEFWTAAEYIPLRDFLQRELIFDGDIMD
>lcl|NZ_CP026520.1_prot_WP_042232729.1_4357 [locus_tag=PC41400_RS21965] [protein=DegV family protein] [protein_id=WP_042232729.1] [location=complement(4852004..4852858)] [gbkey=CDS]
MSRVRIVTDSTSDIPQATREKLGIEMVPLKIHLDGETYLDAVTLDAEQFYHKLVAAHSMPTTSQPSPAEFLALYEKLLAE
DPEVKILSIHLSSAMSGTYQSALLAKSMLDEEDRVQVIDSKSACYGIGALVVAAAEAAREGKSLEECAEIVREIRESFVI
YFLVDTLEYLQKGGRIGKASALIGSLLNIKPILSIDPEGEIASVDKVRGQKKAMARIIEMMRRDIPGGDIHISVAHANNT
EAAEELYRLIEQNFTVKSMQYITIGPVIGTHAGPGTVAAFVSPA
>lcl|NZ_CP026520.1_prot_WP_042232728.1_4358 [locus_tag=PC41400_RS21970] [protein=DAK2 domain-containing protein] [protein_id=WP_042232728.1] [location=complement(4852874..4854643)] [gbkey=CDS]
MSKRFNTINGNDFMQMVFAGADNLRRNVDRVNGLNVFPVPDGDTGTNMNLTLTSGVEELKRKPSAHIGKAAEALSKGLLM
GARGNSGVILSQLFRGFAKYTHDMEEIDVQQFAAALQQGVDTAYKAVVKPVEGTVLTVSKEAAKHAVASARRCADVTELV
EEVLKQASETLSRTPDMLPVLKQVGVVDAGGQGLVCIYEGFVSALNGVQQEDTSSFIVADAPVLGQAAKDKDIATLQLAP
EAHHMDRTGPAQLHMSTDDIEFGYCTEFLLKVVPGKTKGLEFNEIHFRSQLGKHGDSLLVVADDELVKVHIHAEYPGQVM
NLAMEYGDLTRIKIENMRDQHSHILEEAAEVYERAAEEEAQRSVVPAAKELKPYGFVAVALGQGITEIFTSVGVDVVLSG
GQTMNPSTEDVVKAIEQIDAETVFILPNNSNIIMAAKQAQDLVEDKRIIVIPTKSIPQGLASILAFQEKAGADVNEQTMN
EAIQGVRSGQVTFAVRDTSIDGIEIKEGHFIGIEDGKIVVSEPELVEASRKLLDHMIRDAEIVTILTGEDAGDEQTEQLT
AYIEAQYPNVEIELHPGGQPLYAYIFSVE
>lcl|NZ_CP026520.1_prot_WP_009672588.1_4359 [locus_tag=PC41400_RS21975] [protein=Asp23/Gls24 family envelope stress response protein] [protein_id=WP_009672588.1] [location=complement(4854662..4855021)] [gbkey=CDS]
MPIEVSNELGKIHVTDEVIAVLAGSAALDCYGLVGMASRKQLKDGIAELLGRDNLGRGVEVRREQDRVEIDLYIIVSYGT
KISVVAHNVQTKVKYVLNEVVGLHVDEVHIFVQGVRVSR
>lcl|NZ_CP026520.1_prot_WP_009672666.1_4360 [gene=rpmB] [locus_tag=PC41400_RS21980] [protein=50S ribosomal protein L28] [protein_id=WP_009672666.1] [location=4855305..4855493] [gbkey=CDS]
MSRKCYITGKSPKSGNHVSHANNKNKRTWGVNVQKVRILVDGKPKRVYVSTRALKSGKVERV
>lcl|NZ_CP026520.1_prot_WP_036739476.1_4361 [gene=spoVM] [locus_tag=PC41400_RS21985] [protein=stage V sporulation protein SpoVM] [protein_id=WP_036739476.1] [location=complement(4855576..4855656)] [gbkey=CDS]
MKFYTVKLPKFLGGFVKAVLNTFSKN
>lcl|NZ_CP026520.1_prot_WP_042232726.1_4362 [gene=rpe] [locus_tag=PC41400_RS21990] [protein=ribulose-phosphate 3-epimerase] [protein_id=WP_042232726.1] [location=complement(4855915..4856577)] [gbkey=CDS]
MLRIAPSILSADFAKLGEEIREVEKAGADWIHVDVMDGHFVPNITLGPPIVSAIRPHTSLPLDVHLMIEQPERYIADFAK
AGADLISVHVETCPHLHRTLHQIKEHGIKAGVVLNPATPLSAIEHVLEEYLDLVLIMTVNPGFGGQSFIPGMLSKIRDLR
GKLRERGLDHVEIEVDGGINEKTAPLVAEAGATMAVAGNAVFGRPDRAEAIRLIRESAGN
>lcl|NZ_CP026520.1_prot_WP_042232724.1_4363 [gene=rsgA] [locus_tag=PC41400_RS21995] [protein=ribosome small subunit-dependent GTPase A] [protein_id=WP_042232724.1] [location=complement(4856579..4857580)] [gbkey=CDS]
MPQGLIVKALSGYYYVLPEGSRLGSDSLVQCRARGVFKKRGISPLVGDRVKYETTDNGEGTVTEIDPRTSEMIRPPIANV
DTAMLVFSVVEPDLNLQLLDKFLVHIENTGLHAAICLTKADLVDLADDSSEESAAESMEEGEAVQTTASSASAGLVATPM
PDWLTDAVELYRKIGYPVMLTSSRQGEGVDQVAEYLDGKISVFAGQSGVGKSSMLNAIVAGLDLQTNEISHRLGRGKHTT
RHVELISLAKGGYVADTPGFSQLDFVELEAEDLGSCFREFQSYAETCKFRGCLHLHEPGCKVRDAVEQGEIAVSRYEHYL
QFLQEMKEKKRRY
>lcl|NZ_CP026520.1_prot_WP_042232722.1_4364 [gene=pknB] [locus_tag=PC41400_RS22000] [protein=Stk1 family PASTA domain-containing Ser/Thr kinase] [protein_id=WP_042232722.1] [location=complement(4857620..4859767)] [gbkey=CDS]
MIGTKLGGRYEILERIGGGGMALVYKGLDILLHRKVAVKMLRQQYVNDEEFIRRFRREAQAAASLSHPNVVSIYDVGQEG
DLHYIVMEYIEGKTLNEIIKERAPLPVEEAVHYAAQICDALEHAHHNGIIHRDIKPHNILIGKNGRVKVTDFGIARAATS
STITQTGSVVGSVHYFSPEHAKGTNTGEKSDLYSLGIVMYQMLTGELPFHGESPISVALKHLQEDAEDPRRINPMIPQSL
ENIILKAVRKKPDERYQSARAMMEDLETCLQPERRNEPKISFHHDDHDDDENTLVLPAIRPNQQVQIFDDEDDEEESGGP
GKTESAASKSSPPPKKKKKTGLWITILVLVALLLAGAIYFVQGKVVVAEVPVPNVKTDTLQKAQEKLTAAKLGYEVEEVA
DPQPAGIVIDQNPQAGMKAKEQTKVKLKVSKGPELKKMISVVNHNVDDAKAQLIQGLGLKEDQISITYRYSEDQEANTVL
EQTPREGEDLNPAAAAVKLIASKGKETIKMPALVGLSKEEAISRITVSKLKLGAVTEEPSYKVAKGKVIKQFPYEPDNDV
PVGQSIGLIVSSGMPQEAGQMIVNVPLVPSSEGKKSTFKVLVSDAQGDNREYRTETISSPSAVDVKVIVAPDKNAVITVK
EDDKVVSVTTRTYQDFLNQQSGSPSPTPTPPQTGGQTSMQQSGGSKLISSGGEEKGKGKDKDKDKDKDKNKDKDD
>lcl|NZ_CP026520.1_prot_WP_042232720.1_4365 [locus_tag=PC41400_RS22005] [protein=Stp1/IreP family PP2C-type Ser/Thr phosphatase] [protein_id=WP_042232720.1] [location=complement(4859764..4860525)] [gbkey=CDS]
MKMVSVTDTGRVRSVNEDRAVIQHDVGGYSLAIVADGMGGHQAGDVASQMTIDIIREELKSLSSSMTVEECKLAVKEAIR
STNRKVFEYASAREELQGMGTTVVVLLASGQLAILGHIGDSRAYKISGPSILRLTEDHSLVNELLKNGQLTPDEAERHPR
RNVLTRALGTEPDAQPDVQHLSLESGDILLLCSDGLSGLVDDQTILSIVRGESELEAMAQSLLQAALDAGGDDNITIILA
SGDSGPASGGDPA
>lcl|NZ_CP026520.1_prot_WP_174446836.1_4366 [gene=rlmN] [locus_tag=PC41400_RS22010] [protein=23S rRNA (adenine(2503)-C(2))-methyltransferase RlmN] [protein_id=WP_174446836.1] [location=complement(4860541..4861614)] [gbkey=CDS]
MTVKNKTLQVDKPFVYDYSLEDWKQWTEENKEPVFRAGQIFDWLYVKRVTDFEEMTNLSKALREKLKAGFSFVTLTEIAK
QQSKDGTVKFLFELADKNAIETVIMKHSYGNSVCVTTQVGCRIGCTFCASTLGGLKRDLRAGEIVAQIVKAQQLLDATGE
RVSSIVIMGIGEPFENYDAMMKFLNIMIHPKGLNIGQRHITVSTSGIVPNIYRFADEDTQINLAISIHAPNDALRSKLMP
VNRRFPFADLIESCKYYMAKTGRRITFEYALMGNVNDQAEHAEELAEVLKELPMCHVNLIPVNYVQERDYVRTTRDDIFN
FQRILERNKINATIRREQGSDIAAACGQLRAKHMEAK
>lcl|NZ_CP026520.1_prot_WP_042232716.1_4367 [gene=rsmB] [locus_tag=PC41400_RS22015] [protein=16S rRNA (cytosine(967)-C(5))-methyltransferase RsmB] [protein_id=WP_042232716.1] [location=complement(4861931..4863415)] [gbkey=CDS]
MCASPVKKNEPDNRKGQAGGDSGPASRSRTDRRNSQAAKPKREHVKKSARDAALSVLTGVEEAGSYSNLKLNQILREADL
PRPDAALATEIVYGTIQRRNTIDYYLAKLVTKGLGKLQPWVRSLLRMSLYQLLYLERVPDHAVVSEAVNIAKRRGHQGIS
GMVNGVLRSAIRQRAELTLPQGMTDAERISLEHSHPEWLVRRWIKQYGAEACERICKANNEPPHVSIRANARRQHQDALL
EQLRLGGFDAEPSALSPAGIVVSSGGNMALTPGFGQGDYSIQDESSMLVAEALQPEAGMTVLDCCAAPGGKTAHIAEKMD
DTGTVWACDLHPHKKRLIDEQAQRLGLKSVRTLVKDARELAGEFEPASFDRILLDAPCSGFGVIRRKPDLKWGKREEDIA
GIAAVQQSILEAVHGLLKPGGVLVYSTCTIERTENEDAVARFLEAHPEFVFDPIPTTGWPESAPLEQAASGMISIMPDRY
GTDGFFIARIRRLA
>lcl|NZ_CP026520.1_prot_WP_042232713.1_4368 [gene=fmt] [locus_tag=PC41400_RS22020] [protein=methionyl-tRNA formyltransferase] [protein_id=WP_042232713.1] [location=complement(4863557..4864528)] [gbkey=CDS]
MKLIFMGTPDFAVPSLRALIEGGYDVAAVITQPDRPKGRKRVLTPPPVKVEAEKHGIPVFQPEKLRTSDTVDEIRRISPD
LIVTAAYGQILPKSVLDIPRLGCINIHASLLPKYRGGAPIHHAIIQGEQVTGVTIMYMAVGLDTGDMISKVEVPIGDDDT
TGTMFDKLSLAGAELLKRTLPELIAGTARAVPQDDSQAIYSPNINREDEKIDWNRSSKDIWNLVRGLNPFPGAYTTWNGE
ILKVWVSKKPEDLAPGADRTSEGSTGAGLKPGTVIGADEDGIEVATGDGSLKLLNVQPAGKKAMEAAQLVRGSSIGKGTV
LGE
>lcl|NZ_CP026520.1_prot_WP_042232709.1_4369 [gene=def] [locus_tag=PC41400_RS22025] [protein=peptide deformylase] [protein_id=WP_042232709.1] [location=complement(4864531..4865013)] [gbkey=CDS]
MAIRMIVKEGDSVLRETAKQVTKFNSNLHKLLDDMADTMYDAPGVGLAAPQVGILKRVIVMDVGDENGLIEIVNPVVVEQ
SGEQIGPEGCLSIPGLEGEVKRPFKVKVKGQDRNGEEFELEGEELLARCIMHEVDHLNGVLFTDLAIRLYRPELEEKEER
>lcl|NZ_CP026520.1_prot_WP_042232706.1_4370 [gene=priA] [locus_tag=PC41400_RS22030] [protein=primosomal protein N'] [protein_id=WP_042232706.1] [location=complement(4865182..4867758)] [gbkey=CDS]
MFAKVIVDVPAKQTNRAFDYSVPPSLQGWVEVGSRVGVPFGPRVLQGFVVELHETTTVDPKRLKAIRHVLDAAPPLTEEL
VQLGHWMSSTYLCHEVTALQAMLPGALKAKYERFVQASEAGSAGRTELLLLLPAQENILNYLRSKGKAKMEDLLDAFPED
GGLIRQMLDEGQLVETQGMKDRLATKKALTVFPPGAHEASEAATETHDAGTSGNAPGLGGSADLLAIAEELPARSHKQKE
VLRYLHENPRPIRLTELMDKLQVGAGTVKGLADKGWIGLREVEVMRDPYANRHFEETKPLPLTPEQQSVYDDIAATLDSR
SHQVYLLQGVTGSGKTEVYLQAISRCLEQGREAIVLVPEISLTPQMVERFKGRFGDEVAVMHSRLSLGERYDEWRKIMRR
QVRVVIGARSAVFVPFRQLGLIIIDEEHESSYKQEESPKYHARDIAAWRARYNGAAVVLGSATPSLESIQKTKAAPGGKP
EYKLLKMKSRVAGRPLPPVHIVDMREELRSGNRSMFSRLLHRGLEERLLKGEQTVLLLNRRGYATFVMCRTCGYVAGCPH
CDISLTYHQVSRTLRCHYCGYAERQMELCPDCGSEHIRHFGTGTQRVEEELVKLFPGIRVIRMDVDTTTEKGSHEKWLTM
FRNKQADVLLGTQMVAKGLDFPDVTLVGVIAADTVLNLPDFRSAERTFQLLTQVAGRAGRHQLPGEVFVQTYTPEHYSVI
TASRHDYEGFVEEELPHRRKLGYPPYCRLILITLTHEQVPLLVRTAEAVAQRLRELGAAEQENVFASGTGEDFFMNVLGP
VASPIPRIKDRYRFQCVVKYRGEASASRIVAKVLDEFAERTTKDKLTISVDVDPQMLM
>lcl|NZ_CP026520.1_prot_WP_042232704.1_4371 [gene=coaBC] [locus_tag=PC41400_RS22035] [protein=bifunctional phosphopantothenoylcysteine decarboxylase/phosphopantothenate--cysteine ligase CoaBC] [protein_id=WP_042232704.1] [location=complement(4867743..4869020)] [gbkey=CDS]
MLSGKTIVVGVCGGIAAYKAAALCSKLSQAGADVRVIMTESAVQFITPLTFQSLTRKPVAVDTFDEKEAAVISHIDLADR
ADLVVVAPATANMIGKMAHGLADDMLSTTLLATTAPVMVAPAMNVHMYTHPAVADNMRVLQERGVLFVEPGTGQLACGYV
GQGRLAEPEEIFARIAAYFQPDTPLQGRNVLVTAGGTVERIDPVRYITNDSSGKMGYAIAEAAKRLGANVTLVSGPSALP
VPAGVERISVTSALDMYDKVMARFEKTDLVFKAAAVADYRPAEQREQKMKKSGETLTIELVKNPDILQALGEAKTRQFVV
GFAAETERVDEYAQGKLLRKKCDLLVANDVSMQGAGFGTDTNVVRIFDAGGLVEAPPMMTKAAIAEHIVRLAASRQQMRQ
QGAETPEPDAEEPGADKGETPCSPK
>lcl|NZ_CP026520.1_prot_WP_042232842.1_4372 [gene=rpoZ] [locus_tag=PC41400_RS22040] [protein=DNA-directed RNA polymerase subunit omega] [protein_id=WP_042232842.1] [location=complement(4869185..4869400)] [gbkey=CDS]
MLYPSIDKLLDKVDSKYSLVVAAAKRARSLRDGTNTHLQDIKSHKQVGVALEELYGDLVKYEKIETKEDAE
>lcl|NZ_CP026520.1_prot_WP_042232839.1_4373 [gene=gmk] [locus_tag=PC41400_RS22045] [protein=guanylate kinase] [protein_id=WP_042232839.1] [location=complement(4869410..4870042)] [gbkey=CDS]
MTSNTNTLERDRGILIVLSGPSGVGKGTVCSALRKISPELVYSVSTTTRKPREGEVEGVNYFFKSKDQFEQLIASDEMLE
YAEYVGNYYGTPRRFVEETLASGKDVILEIEVQGALKVKEKFPEATFIFLIPPSLAELENRIVTRGTETNDIIRSRMSVA
VDEIRLMRHYDYGIVNDEVHKACSKIQAILMAEHCKCDRILPRLEQWIGE
>lcl|NZ_CP026520.1_prot_WP_009672664.1_4374 [locus_tag=PC41400_RS22050] [protein=DUF370 domain-containing protein] [protein_id=WP_009672664.1] [location=complement(4870032..4870292)] [gbkey=CDS]
MAIKLINIGFGNIVSANRIISIVSPESAPIKRIIQEARDRHMLIDATYGRRTRAVIITDSDHVILSAVQPETVAHRLSNK
DDDHDE
>lcl|NZ_CP026520.1_prot_WP_042232702.1_4375 [locus_tag=PC41400_RS22055] [protein=YicC family protein] [protein_id=WP_042232702.1] [location=complement(4870324..4871208)] [gbkey=CDS]
MIRSMTGFGHASLSSQGYRVSIDIKTVNHRYCDISVRMPKEWMAYEDTIKKTVLRFIKRGRADCFVTIERDSASNKSVTV
DWALVDGYMQAAEQLRQKHGLTGELHLSDLLKLPDLLAISEQREGNDDALADLLVACAEGALAELNGMREKEGSHLSEDL
TVRLTMIEDLNVEMKRYAPEVVREYAVKLRQRIQELLTDQVPADEQRLAAEIALFADRSNVDEEMTRLHSHISQFRQLLG
SREPVGRKLDFLIQEMNREANTIGSKAGHIEVSGRVIELKAELEKMREQIQNIE
>lcl|NZ_CP026520.1_prot_WP_042232700.1_4376 [locus_tag=PC41400_RS22060] [protein=bifunctional homocysteine S-methyltransferase/methylenetetrahydrofolate reductase] [protein_id=WP_042232700.1] [location=complement(4871241..4873133)] [gbkey=CDS]
MKPELRQVLAQKILIGDGAMGTYLYQQGFPVGISYEELNLIQSEVVADVHRRYYEAGARLIETNTFSANREKLSKYGLEQ
EVEAINRAGVELARSSVGQDAYVVGAIGPIRAGMRKNVRTADVEDALREQILVLLDTQVDGLLLETFYDLEEMLIALRLI
RQLSSIPVICQFATEGISTTHDGISLEDAFLRLKEEGADVIGFNCRTGPNGILRSLQGVAQLSDIPFSVFPNAGIPDYVD
GRYSYAATPEYFAESALRFADLGARIIGGCCGTTPEHIAAMVKALQGYEPNRSEVYAAESVSAAERAVVTEAAPQTSVRE
ADVEPTLVDIVKQRHAVIVELDPPRDLDIGKFMEGTKALQDVHVDAITMADNSLAVTRMSNLALGMLVKEQHGARPLIHI
ACRDRNSIGTQSHLMGLHALGIDHVLAVTGDPARFGDLPGASSVYDMTSFEMIRMIKQLNEGIAFSGKPLKKKANFIVGA
AFNPNVKYLDKAVQRLERKIESGADYIMTQPVYDPKLIERIHEATKHLDVPVFIGIMPLASGRNAEYLHNEVPGIQLSDE
VRARMTGLEGEEGRKEGVAIAKELLDAAMPLFNGIYFMTPFLFYDMSVQLTQYVWQKAERCELLLSPLLK
>lcl|NZ_CP026520.1_prot_WP_042232699.1_4377 [gene=dapF] [locus_tag=PC41400_RS22065] [protein=diaminopimelate epimerase] [protein_id=WP_042232699.1] [location=complement(4873243..4874079)] [gbkey=CDS]
MEFTKMNGLGNDFIVLSGYKELPDNVAELAVRWCNRFFGIGADGLVFILPSEKADVQMRIINSDGSEPEQCGNAIRCVAK
YVYDRDPGKREAITVETRGAGVQQVHVTAEKGIAKAIRVDMGEPVLQGLEIPTTIDAERVIDQPIEAGGRTFTFTAVSMG
NPHCVIDVEDAPSFDLAAWGPLLEQHPLFPNKANVEFVTVRSRDYADMRVWERGAGPTLACGTGACATLVALVLTGKTDR
TATISLKGGDLLIEWNEADNRVYMTGPAAEVYKGKVEE
>lcl|NZ_CP026520.1_prot_WP_042232698.1_4378 [locus_tag=PC41400_RS22070] [protein=calcium-translocating P-type ATPase, SERCA-type] [protein_id=WP_042232698.1] [location=complement(4874240..4877020)] [gbkey=CDS]
MSQKEWYQMTADEVLQSQRVHPSEGLSSAEADNRQTEAGRNELSEGKSVSPVTLFLNQFKDFMVLVLMGATLVSGLLGEY
LDAITIVAIIIMNGILGFIQEFRAERSLRALKELSAPGAKVMRGGELQTIPARDLVPGDIILLESGDRVPADIRLIEANS
FYVEESALTGESVPVGKTVDPLSSDTVTIGDQRNLAFMGTMVTRGTGKGAVVRIGMETEMGKIADLIQNTQTLETPLQHR
LEQLGKILIIVALCLTVMVVVAGIVHGQPPYAMFLAGVSLAVAAIPEGLPAIVTIALALGVQRMIKRKAIVRKLPSVETL
GCASVICSDKTGTLTQNKMTVTHLWVGGSLLEVSGDGYTPEGEISAGGTRVNPAKNSMLRQFLQVSALCSNAVLYKEEAE
PVKKKRAKDEPPETVWNVKGDPTEGALVVLAAKAGVTHEVLNPQFDRLAEFPFDSERKRMSVIVSSGGRKLVMTKGAPDV
LMQHCTYILWDDKVIPFTSTLKAKVMAANEGMARSALRVLGTAYRELKPTESCEDHEDAERGLVFVGLAGMIDPPRREVR
EAMAKCRKAGIKTVMITGDHLTTAEAIAKQLGMLPAGGMCISGHQLAAMDDEALESKVDDIYVYARVSPEHKLRIVKALQ
AKGHVVAMTGDGVNDAPAIKAADIGIAMGISGTDVTKEASSLVLSDDNFATIVAAIEEGRGIYENIRKFIRYLLASNVGE
ILTMFLAMMAGLPLPLIPIQILWVNLVTDGLPAMALGVDQAEKDLMQHKPRSARENIFARRLGWKIISRGFLIGVCTLGA
FWLVLRVNPGDAQHLVKAQSVAFATLVMAQLIHVFDCRSSRSIFHRNPLQNKYLVLAVLSSLVLMLGVMYTPQLQPIFKT
VPLGFKDWIIVLVAAGIPTFLMGLGSVMQNPSKKKTAKYSSKPSFR
>lcl|NZ_CP026520.1_prot_WP_042232696.1_4379 [locus_tag=PC41400_RS22075] [protein=NFACT RNA binding domain-containing protein] [protein_id=WP_042232696.1] [location=4877195..4878952] [gbkey=CDS]
MALDGLVLHAIVHELQECVGGRINKIHQPTGSDVVMQIRARGRSLKLLLSANPTYPRMHVTEGQYLNPQEAPMFCMLLRK
HCEGGIIDSIEQVGLERIVHFRVRQRDELGDVSTKTIVVEIMGRHSNLILMDPSTGTILDGIHHVTPAISSYRIVMPGSA
YAEPPAQDKANPLDTSEAVFRRSMSAGIGEDAAPEKQMLASFSGLSPLAAREIVYRSGTRLGVPYGEDALAPLWQSFRSV
TDAVRERRYSPVIVEETKTGKLFFSVMELTQIEGESRLFPTVSECLESYFGDKAERDTVKQRAADLVKLLQNEKAKNIKK
MDKLRETKEEAKDADRFRIMGELLTAHLHQIRKGDKEAVVVNYYEEAQPQQSIALDPLLTPSENAQRYFKRYTKSKNSLA
YVEEQMSQTEMEIQYLSSLLQQLGSASMRDIAEIRDELAEQGYVRERHTKGKRKKKTDKPSVACFTSSEGVPIYVGKNNT
QNDYVTNRLASAGDTWLHTKDIPGSHVVIRSQSFGEATLHEAAQLAAYFSQAKESSSVPVDYTLIRHVHKPSGAKPGFVI
YEQQKTLFVTPDDQLIKQMPQTVKA
>lcl|NZ_CP026520.1_prot_WP_042232694.1_4380 [locus_tag=PC41400_RS22080] [protein=MerR family transcriptional regulator] [protein_id=WP_042232694.1] [location=complement(4879054..4879986)] [gbkey=CDS]
MSRLYTIKEASMRSGLSTQLIRKWEERYGAVQPDRFPNGYRGYTNHDIETLIWLKSKADSGVPIGLAVQEKQLRLESSAS
TETISRSEAELPFIHSELKEYQEKLLGLFLQLDQAGAQRFFDRLAALHQMDFILLQVLQPTLIRVGEMWERGEISEYQEH
FGSHFIRERLLALKNLFSIPPGGPRLITACSPYESHELGILFLGYFALQNGFQTLYLGASPSEKGILDCLEQIRPAAFAF
GSSMKSMLDEAYLFYAKLDRRIGELGSRTKVFIGGNAIERDEVLHGTKHIYLLSGDARECALKLKKLTAY
>lcl|NZ_CP026520.1_prot_WP_042232692.1_4381 [locus_tag=PC41400_RS22085] [protein=UDP-glucose/GDP-mannose dehydrogenase family protein] [protein_id=WP_042232692.1] [location=complement(4880061..4881377)] [gbkey=CDS]
MERIAVIGTGYVGLVTGCCYAEIGHEVICVDIDPDKVRRLSLGEIPIYEPGLGELAASNQAAGRLSFTTDLARAVQDSSI
IFIAVGTPTLENGEVDMQQVRGAVETLARQMNDYKIIVMKSTVPVGTSRRIRQWMEEGQQRPVPFSVVSNPEFLREGTAI
HDTFHPDRVVIGSDDADAALRVERLQAAFGGEVVRTDPESAELIKYASNAFLAAKISFINEMANVCEKVGADVSLVARGM
GLDNRIGPKFLQAGIGYGGSCFPKDTRAQLKIAQNADYDFRILRAVIEVNTLQRERFVEKVTAALGGNATGRSITVWGLA
FKPNTDDLRDAPALDIIRALQERGAVVTAYDPAAASKASALLPGVRCLTDPYEAAENAEAVVIATDWEDLRRIDFDRLRS
LVRKPLIIDGRNMFAPAEMAEKGFTYVSVGRPAFPGPV
>lcl|NZ_CP026520.1_prot_WP_042232690.1_4382 [gene=galU] [locus_tag=PC41400_RS22090] [protein=UTP--glucose-1-phosphate uridylyltransferase GalU] [protein_id=WP_042232690.1] [location=complement(4881393..4882274)] [gbkey=CDS]
MKIRKAIIPAAGLGTRFLPATKAQPKEMLPLVDKPAIQYIVEEAIESGIEDILIVTGRNKRAIEDHFDKSVELEMMLEEK
GNDELLSLIRTVTNLADVHYIRQKQPLGLGHAIYCARKFIGNEPFAVLLGDDILRSSKPALRQMMDVFEETQTSVIGVRE
VPWEDVSKYGIVVPSDEITAYKKIADLVEKPDRGDAPSNLAVIGRYLIMPEIFDILEKCEPGRGGEIQLTDALRILNEKQ
PMVAYPIEGRRYDVGDKFGYIQATIEMALEREGLQEELRSYLTNLVRHEFHIG
>lcl|NZ_CP026520.1_prot_WP_053228761.1_4383 [locus_tag=PC41400_RS22095] [protein=PHP domain-containing protein] [protein_id=WP_053228761.1] [location=complement(4882261..4883118)] [gbkey=CDS]
MTQWADLHTHTTASDGTQAPEDNVRLAKEAGLAAVAITDHDTVSGVRAAQLEGRRIGIEVVPGVEISTVAGGQDIHVLGY
YIDTESESFLERLEELRNTRNKRNEMMLARLNEMGLAVSMEEVVKHLEAGKSEEDTIGRPHIANVLLAKGYVSSMNEAFE
KYLGKGGAAYINPPRIRPVTAIDWIRDAGGVAVLAHPGLYGDEDIVRELIAYGLDGIEVNHADHTPEQEAHYGRLAREHG
LIATAGSDFHGVRAGQVFHAPLGSKRTSLETVRKLQERRNNRENS
>lcl|NZ_CP026520.1_prot_WP_042232837.1_4384 [locus_tag=PC41400_RS22100] [protein=selenium metabolism-associated LysR family transcriptional regulator] [protein_id=WP_042232837.1] [location=complement(4883189..4884100)] [gbkey=CDS]
MALNFHQLHIFYTVAEKGSFSHAAQALHMTQPAVTMQVQSLEDHFGIKLFHRSTKKIELTEAGRTLIPYARKCVELIRET
ENAMTGFTAMAEGRLQLGASLTMGEYILPRLLGPFRKEYPNISVAMKVMNTTQILDEIFAHQLTFGLVEAPIQHPDVHTE
AILSDELKLIVPAGHPLAEMETIRMEDVFRYPFVLREEGSGTRRVMEEELERTGISCSGMDIVMELGSTGAIKSAVEAGL
GVSILSQSSVKHEVRLGILKIKEIEGVSFSRSFYAIYLNSTLLPLSAVSFLNFMRRDDLVKWL
>lcl|NZ_CP026520.1_prot_WP_042232685.1_4385 [locus_tag=PC41400_RS22105] [protein=hypothetical protein] [protein_id=WP_042232685.1] [location=complement(4884217..4884654)] [gbkey=CDS]
MRDKTMDRWSTYEPFYVNLNDKKIADIVITNHARLRWTDRVESEKTGFEDICDFLWDKLKDGRIASYYKNEQDVYLVDDD
LVMVAEFSVIENETDIAGNPLHKMIIVTFLGRMSETIELRDLKSYYSWLRHSRRMTLIKNSRKRR
>lcl|NZ_CP026520.1_prot_WP_009672616.1_4386 [locus_tag=PC41400_RS22110] [protein=YlbG family protein] [protein_id=WP_009672616.1] [location=complement(4884954..4885217)] [gbkey=CDS]
MITERTGLIIWVTDLKAAKNLERFGSVHYMSKKMHYVVLYIHASRYDETVRQLQKLAYVRKVERSYRGEIKTEYESHGPD
KSKFYTL
>lcl|NZ_CP026520.1_prot_WP_042232682.1_4387 [locus_tag=PC41400_RS22115] [protein=asparaginase] [protein_id=WP_042232682.1] [location=4885291..4886301] [gbkey=CDS]
MSASSAVIARVIRGPITESIHRGHIAVVDHTGRLLHSLGDPQAVTFARSTAKLIQALPVLESGAASDFGFTDAEIALICA
SHNGEDRHVRAARSILAKAGAEADDLQCGSHYPFHKPTSRQMRQEGEAPTPLHNNCSGKHSGMLSLAAKLGVSTEHYMSP
EHPVQRQMLQAVAEMAGLAPEEIPLGVDGCGVPVFGLAIDRLALAYARLGKPDGLSAERADACRRIIAAIRREPSYLAGS
DRFDTRLAEATGGRIIGKMGAEGLFAVTVPDEGLGLVLKIEDGAQRALYCAMFEALVQLGLLHTDEMEALAEFREPKVTN
WQGTVVGSIQADFKLI
>lcl|NZ_CP026520.1_prot_WP_042232680.1_4388 [locus_tag=PC41400_RS22120] [protein=YlbF family regulator] [protein_id=WP_042232680.1] [location=complement(4886458..4886910)] [gbkey=CDS]
MSVAEAKTMDMSSILMGAYQVGDLINNSAEVADYLYWKQAVEQDFEAQRLVREFNKAKDLFEECERFGHFHPDYHAALDK
VQAVQSRLEQVESINRFKEAEERLDQLLYTISKTIATAVSDTIKVPGGLDTGSGGGCSSCGSGGSCSSCG
>lcl|NZ_CP026520.1_prot_WP_042232677.1_4389 [locus_tag=PC41400_RS22125] [protein=helicase-associated domain-containing protein] [protein_id=WP_042232677.1] [location=complement(4887135..4889141)] [gbkey=CDS]
MNYAQILDKMPPAASRQLLGQDWIKSAAGHGIEAREAFGNEDVLRKLRAALQPDEEKLLRLIVVRFGSRPFTPAELEKET
ADWAAALVKTALTGLRRRGLIAAFRKSWGDRLFLLPQDGLRGWQAVFFGPGCDVVGHGDGAGSQRNAGSDYNARLRQLYE
QGDKPRQGLALDLFLLLNYADLHELPLTQRGALHKRHVQRMTALLGFDETDLAGCGLQFAHADTYPVSLAAALDFLQRLG
LLHRIRDRLLLHKPRLRDWLATDEARQSEMLYRLWTEHRFPEDVDLQHAVCLLDKLAAGTKIRLSRLAAELNRRETLAGG
EEEGARFSLGLVQKWLQPLRAFGWLDWEPEDSGDVSVSWLHPVRYEQRANEERKTAKPSAFLYVQPDFEVMIPPECPFTV
RWEIARFAELVKSDRMAVYRLTKESVQRGADHGTPPEQALELLESHARHGVPETVRLTIAAWGEQYGRVVLGGVTLLRCA
DAQLAEQVSRHPKIGRLLLEKIGDSDFIVPAGKMSDMMKLLDAAGFSPRQAADFQDEEEAPDPDGDPETETLKGIVYDQD
PVQYLELETRLPTAEEMYPGMRDIPAVWYRESRGYHASTILKIVRQAVTWGAPVLAEVEGGAVAVWIPRRAVERGEELTI
ELSDGAGGNRTVQAGRISRIRILLPGVS
>lcl|NZ_CP026520.1_prot_WP_129112525.1_4390 [locus_tag=PC41400_RS22130] [protein=helicase-associated domain-containing protein] [protein_id=WP_129112525.1] [location=complement(4889116..4890813)] [gbkey=CDS]
MNPASDNALIVQQDLTVLAEADHERFESIRAELALFADLVKSPKHLHTYKITPLSLWNAAASGLRGADIIRFLKEESKYG
ISRRMEDELLSILGRYGRLRLISDKGKLYLQSADTELLARLGGLKELKRFFDREDAGPAEAVQSSALAVRPDMRGVMKQE
LMRLGYPVIDLAGYHAGEDLPAELKGVELRDYQRQAVECFHPADGSDGGSGVLVLPCGAGKTVIGIAALTKLSCAALILT
TNVTSVRQWIKEILAKTSLDESRVGEYSGDVKLVRPVTIATYQILTHRREKGGEQAHMELFNSRDWGLIIYDEVHLLPAP
VFRATADIQATRRLGLTATLVREDGREEDVFSLIGPKLFDMPWKRLELEGWIARVTCTEIGVEMETGELQNYYEADKRSR
FRIAGENSRKFPALTRLLARHDGEAILIIGQYLDQLKEVARKLTIPFISGEMPQADRQRLYEAFNSGLVRILAVSKVANF
AVDLPDASVAIQLSGSYGSRQEEAQRLGRILRPKKGRNEAFFYTLVSRGTSEQEYAIKRRIFLLEQGYEYRIEEGGSDEL
CANTR
>lcl|NZ_CP026520.1_prot_WP_042232671.1_4391 [locus_tag=PC41400_RS22135] [protein=thermonuclease family protein] [protein_id=WP_042232671.1] [location=4890984..4891754] [gbkey=CDS]
MNTKKALLLFILAALFLAACGSAQPKTDSGSFKETVLARYPELKTQKMTTAKVKRVVDGDTFELENGDKVRLIGCNTPET
VKPNSPVEPYGKEASDYTKKRLTGATVYLFNDAGDTDKYGRLLRYVFIEGESVMYNELLLQEGYANVMTIQPNVMFSDRF
VKTEREARSKNKGLWGAKSASTPSKTGDSPKGGSGKNSEDANQTAACPDPQIKGNINSKKEKIYHVPGSSSYEQTQAEAM
FCTENDAKKAGYRKAG
>lcl|NZ_CP026520.1_prot_WP_042232669.1_4392 [locus_tag=PC41400_RS22140] [protein=CBS domain-containing protein] [protein_id=WP_042232669.1] [location=complement(4891821..4892255)] [gbkey=CDS]
MPTVREVMSTDCVTVTLQDNVYEIALKMKQHDIGFIPVVEGNTLIGVVTDRDLVVRGYAEKHSGSTAVKQVISEHLTTIS
PETDVQEAAKVMAQHQVRRLPVVEDGNLVGVIAIGDLAVRDKLEDEAGEALSRISKGRSTSSNK
>lcl|NZ_CP026520.1_prot_WP_042232667.1_4393 [locus_tag=PC41400_RS22145] [protein=M20 family metallopeptidase] [protein_id=WP_042232667.1] [location=complement(4892306..4893544)] [gbkey=CDS]
MKGRNVSNTGAGDTLIIQGEELDKEWEAMYETMVSWRRHLHQHPELSYKEVNTAAFVAEKLTEWGLDVRTGVGGYGLIAD
LQGSAPGPTVALRADMDALPIQDEKQCGYASKVPGIMHACGHDAHTSTLLAAAKIWSTKKERLKGRIRFIFQHAEEVTPG
GAASMIEAGALDGVDVVYGVHLWTPLPVGAVGSNPGAMMAAADEFHFEIRGKGGHGGMPHQAIDSVVIGSHAVVNLQTIV
SRTVSPIESCVVTIGSINGGTNFNVIAETCKMKGTTRTFDSALRLQVKERIEEIVASTCKMYGAESVMDYRLGYPPLVNH
PGEFERFLEVASGMLPEDSVLTIEPVMAAEDFAYYLQQVPGCFIFVGAGNAQTGADYPHHHPKFDLDERAMLTAGKLLTR
MALHVLDETGTR
>lcl|NZ_CP026520.1_prot_WP_009672640.1_4394 [locus_tag=PC41400_RS22150] [protein=YugN family protein] [protein_id=WP_009672640.1] [location=4893681..4894025] [gbkey=CDS]
MIIETKGFKGLKSDLAHLDEVTTKLGFVRGQWEYTRATYDLKMEDPTTKSDYYLRVNTRAESGKLESPYAILYIEEAYIG
RATFPHGLDYESPIPDAILNKAKQKLSQLQQLLA
>lcl|NZ_CP026520.1_prot_WP_042232665.1_4395 [gene=ftsW] [locus_tag=PC41400_RS22155] [protein=putative lipid II flippase FtsW] [protein_id=WP_042232665.1] [location=4894038..4895219] [gbkey=CDS]
MSTPRRGTPDFLLLFLTFALVCFGIVMVFSSSSVYASTKFSNPWYFATRQILFAVIGTFLMLFCMNIRYSKLKNWVKPGF
LVVVAMLVAVLFFKGINGSRSWLPLGTFSLQPAEFAKLAVIVYLAQLISKKGDNIQNMKKGLMPAVLITAFIAFWILLQP
DVGSAAILILCASVVIFVGGARVKHLFSLGVAGGLAAAVFLGFYFLFNFKSASDHVGLRMKRFTTYLNPKDDVLDSGFQV
MQSLYAFGHGGITGAGFGQSIQKLYLPEPYTDFIFAIIGEELGMIGSILFLLVYIIFLWRGLVIAVRCPDRFGMLTGVGI
MTMIGVTAIINIGGVTGTLPMTGVTLPLISYGGSSMMVTLISMGIMLSLSREQNRLEKKEKVEKVTQTTRVYR
>lcl|NZ_CP026520.1_prot_WP_042232663.1_4396 [locus_tag=PC41400_RS22160] [protein=Asp23/Gls24 family envelope stress response protein] [protein_id=WP_042232663.1] [location=complement(4895333..4895719)] [gbkey=CDS]
MVEEIQTGNIRISDDVVATIAGLAALETPGIAAMSGGISEGFAKRLSGKNVQRGVSVEVGQVEAAVDLRVIVLYGSKIQD
VCRELQLNVREAVENMTGLRIVEVNVKVEGVSFKEDEAASIDDPTRVK
>lcl|NZ_CP026520.1_prot_WP_042232660.1_4397 [gene=cax] [locus_tag=PC41400_RS22165] [protein=calcium/proton exchanger] [protein_id=WP_042232660.1] [location=4895918..4896961] [gbkey=CDS]
MRKNLFNIGLIVTFALSAVAHYMHWGTTIQFAISCLAILFVAGFLGKATESVAHYAGDRMGGFLNATFGNAAELIIAFFL
VKDGMFDVVKASLTGAIIGNLLLVLGASVLAGGLKFKEQNFNIKLASHNASLMTLAVIALFIPAVFAKSLTQKETTQMSL
IVAGVLIVAYILWLIFSMVTHKKELSDQVVEHSEPAWSKRTSILFLILATVMVGFESEWLVGTLEEFTSRFGLSELFVGA
FLIAIIGNAAEHSAAVMMALKNKIGAAVEIAVGSSLQIALFVAPVLIFSSFFMGNTMDIVFTTYELVAIGVAVIISKSIS
QDGSSNWYEGVLLLAVYLILGVVFFLV
>lcl|NZ_CP026520.1_prot_WP_042232658.1_4398 [locus_tag=PC41400_RS22170] [protein=YlaN family protein] [protein_id=WP_042232658.1] [location=complement(4897010..4897294)] [gbkey=CDS]
MSSSDLLVSLNQKALNLLEEDAYKIEKLIEIQMENLTTRQCPLYEEVLDTQMYGYSREIDFAIRAGLIPANVGKELLNKL
ERNLAKLYEALNKK
>lcl|NZ_CP026520.1_prot_WP_042232655.1_4399 [locus_tag=PC41400_RS22175] [protein=HPr family phosphocarrier protein] [protein_id=WP_042232655.1] [location=complement(4897440..4897688)] [gbkey=CDS]
MPSANNAAIVEISQTANQFRSSIVLQYDNKYIDVKSILGLFTTLLSSGSYELHVHGADADEAKKAMAEVFAKHQLAIQIV
ED
>lcl|NZ_CP026520.1_prot_WP_042232653.1_4400 [locus_tag=PC41400_RS22180] [protein=aminopeptidase] [protein_id=WP_042232653.1] [location=complement(4897792..4898907)] [gbkey=CDS]
MRDPRIQQLAANLVQYSLDVKPGENVWIEMYGSERELVRCLIEEVYARGGKPYVDLIDRRVLGSLIKSATKEQMQEWGEA
DLARMKRMDGYIGVRAGENVNEWADVPEEQMKLYSLYYQKPVHLEQRIKHTRWVVLRYPNASMAQLASKSTDAFEDFYFD
VCNLDYGNMDAAMDPLKALMERTDKVRLVAPGTDLTFSIKGIPAIKCSGQNNIPDGECFTAPVRDSVNGTITFNTPSIQS
GVTFDNISFHFENGKIVEATSSDTARINEVLDMDEGARYIGEFSLGFNPYIQHPMKDILFDEKIDGSLHFTPGQAYEEAD
NGNRSSLHWDLVLIQRPEYGGGEVWFDDVLIRKDGRFVIPELEGLNPENLK
>lcl|NZ_CP026520.1_prot_WP_042232651.1_4401 [locus_tag=PC41400_RS22185] [protein=TVP38/TMEM64 family protein] [protein_id=WP_042232651.1] [location=complement(4898949..4899653)] [gbkey=CDS]
MKWLTFIRTKRFIYGMTIVLVIIAVLSILYRKSDWRHMGASMEQLTPWLISLGMLGKTLGIALVFLQTLFPFVPFVIIAG
ANVALFGIKWGFVVNYVMSVLSSVTAFIIARYFAHDWVERKLSSKPLVQQFNKKLEHHGFLYILMGRLIPVIPSSAISFG
AGVTSVTFRQFVLGTTIGKIPMVLLESLIAHDLFHFKAYKGRLFVLLAIFVILMVLGSLFKKRLDARKQDKPSS
>lcl|NZ_CP026520.1_prot_WP_042232649.1_4402 [locus_tag=PC41400_RS22190] [protein=diguanylate cyclase] [protein_id=WP_042232649.1] [location=complement(4899812..4901827)] [gbkey=CDS]
MSDKLHHPLPYYPVENAKKLDESSLEPYVLHLDRQGGVLSQRDELHQLFKDWIGRSARYLRSFNGSLELFDSFGQNIGFY
TAENFLHPLYRDGTAWTEELLERTAFSQCKRHQRLVTVDDSTYNEGALRGCVTCAAPLYNPSGECVGTIGILAGPVQDHA
LLEALLESQILSLQQFSMSGHSARTALSLSGQINRNERELRKWEMLFEMTNKLHAQIDVNAVLTVVIDCIQELYPFVMLD
LLLSQDHHNGALPFKLLRYQSGEEDICAKAFLEGELFVYEENLGDGIRRREIAAPLSGNQGVYGVLHLKYTCDTVDPAEL
SFITLLADAAGTAFENAKLYEHSNLLVNELRLINDMTQKLTQSLSLHEIFDFACGELTDIFKGEFCCILQKEEEKERLVV
KASNFNRLHYEHLEPGTGYSGLVYDRKEAIIVSDYDPRFHVSSKLMELSKSKSLMAAPILGGGEVLGVILVAHREPHYFS
YDNFKLLQALCSHIGLAITNATLHAELQRLVIMDQLTGLYARHYLNNQVQEMQKKDFCGSLIVVDIDYFKQVNDTYGHQV
GDQILMQVSGILASSIRDTDIAARWGGEELAVYLPKVGKQQALRVAERIRERVMNETRPGVTVSCGVSDWFWEDDKISVE
SLFYKADMALYQAKHDGRNQIVMDQSRSFTS
>lcl|NZ_CP026520.1_prot_WP_042232646.1_4403 [gene=rpsD] [locus_tag=PC41400_RS22195] [protein=30S ribosomal protein S4] [protein_id=WP_042232646.1] [location=4902145..4902750] [gbkey=CDS]
MSRYTGPKFKLSRRLGISLSGSGKELKRPFPPGQHGPGQRKKLSNYGMQLQEKQKLRLMYGVHEKQFRNLYSFAAKQKGI
TGETFMQLLESRLDNLVFRLGFSNSRAGARQLVAHGHVTVNGKKVDIASYSVKTGDVIGLREKSRGLSVIKEALANRNYL
PAYVEFNEAALEGKYVRLPERSELPQEIDEKQIVEFYNRLS
>lcl|NZ_CP026520.1_prot_WP_042232834.1_4404 [gene=tyrS] [locus_tag=PC41400_RS22200] [protein=tyrosine--tRNA ligase] [protein_id=WP_042232834.1] [location=complement(4902848..4904107)] [gbkey=CDS]
MKDVWEKLSAEQKAEAERQVEILRRGAVEIVPEEELKLKIAESLADNRPLKIKLGLDPSAPDIHVGHTVVLHKLRQFQEL
GHHVQLLIGDFTGRIGDPTGKSETRKQLTEEDVKRNAETYKKQIFKILDPEQTTVCYNSEWLSPLNFADVVELSAKLTVA
RVLERDDFSKRYTSGQPIHVHEFFYPLMQGYDSVALKCDVELGGTDQKFNLLMGRTLQKEYGVPAQVAIMMPLLEGLDGV
NKMSKSLGNYIGIDEAPNEIYGKAMSVPDELMLKYYELATDLSHDELSALRTGIEDGSVHPRDAKMRLAATFVRMYHGEE
AASEAERHFVTVFQQRALPDDIEEVELAAGELEDGGIRLVKLLVTLGLQATNGEAKRSIQQGGVRLNEEKVTDPNASVEV
KDGDILQVGKRKFAKIKLA
>lcl|NZ_CP026520.1_prot_WP_042232644.1_4405 [locus_tag=PC41400_RS22205] [protein=transglycosylase domain-containing protein] [protein_id=WP_042232644.1] [location=4904633..4907713] [gbkey=CDS]
MAKKLSFLQNPGVRKTLKVTFHTVKWLFVAFLVVGIMGGAAAFGYVSALVKNEPVRNKEEMRKKIEENAITGFVYFNDDA
VIGQLRTEEDRRLAELAEIPQVVLDAVLAIEDNDFYKHNGVDLKGLSRAVTQKLLNEDVQTGGSTITQQLTRRVFLTLDR
DAGRKAKEILLSLRMERLLSKDEILLAYLNKIPYGNGATGYNLFGIKAAAKGVFNIDELNQLNVAQAAYLAGLPQQPSNY
SAYTSSGAPNPTGLKNAIARQKLVLKRMREENKINEAQYNEALAFDIEASLAKRQQKAYASYPYLMIEAEKEAVKLLLKQ
EHPDLDPEKDAAAYSEASKAMHTQLLRGGYQIYTSIDKTIYDSMQTIGKEAKNYSPTDSKKGEEQVGAVMMDSKTGAVLG
MIEGRDFNKSQMNNAVQAVRQPGSTMKPIAAYIPAMESGKIQPASVIDDSPVILKDGARGYHIPENWNHKYSGLITARKA
LNQSYNIPAIKLFTDTVGIKEAWTFAKKLGITTIEKDDYNAQTGVIGGLKYGVTVKDMTGAYAAMSNKGLYNEAFMIRKI
TDSTGKVIYEHEMKPTRVFSEETAYLITDMMRTVITEGTATSIKKSFKFYGKIPVVGKTGSTQDDADAWFMGYTPNITVG
VWAGYELPVNKLSKAGTQRAMNMWSLIMNTAVEKKPELFPDTSFNRPSTVVQATVSDVSGKLPSDLNRQSGHVVTDWFNR
KYIPKTVDNSMKSMSTIEYNGINYIAQDQTPGDMVKSKVVIVREKSVNSLLKELSAVLNKLPAKNRRPLSFYKPKDYDED
APSEIDPRKDDGSDPNPPSKVAATKSGGTVVITFQPSGNNDVVGYRLYRSDNGGPYIRQNGKVVMTGSDSKFTDSVSPSS
AQSYYVTAVDVAGRESAPGKASSTGSGSSVDSGLVPGDGGSQNGGNGGSDQESETKQTPSSPTGLSVKSSGSGLKLQWNE
NSSKDKVKRYNVYYAEKADGSYKKIDAALSAEYANSSVKHDGYYKVSAVNDAGESRLSSAVKYAAR
>lcl|NZ_CP026520.1_prot_WP_042232643.1_4406 [gene=acsA] [locus_tag=PC41400_RS22210] [protein=acetate--CoA ligase] [protein_id=WP_042232643.1] [location=complement(4907914..4909638)] [gbkey=CDS]
MNQLKGEVVDAVSQNGNLGDYEKARESFNWEQIESNFSWFDTGKVNMAYEAIDRHAESSRKDKVALYYSDQKRDESYTFS
QLKTTSNQFGNVLRKLGVNKGDRFFIFMPRTPELYMALLGAIKIGAVVGPLFEAFMEMAVRDRLHDSEAVAIITTPALVH
RIPTAELPNLKHVIVVGEDLELQEGQVDFHKEMAQASEELDIEWVDREDGLILHYTSGSTGKPKGVFHVHNAMVQHYYTG
NIVLDLKEDDVYWCTADPGWVTGTSYGIFAPWLNGATNVIRGGRFSPQDWYNTIQKYGVTVWYSAPTAFRMLMGAGDDVV
HQFDLSSLRHVLSVGEPLNPEVVRWGLKVYGQRIHDTWWMTETGGQLICNYPSMPIKPGSMGRPIPGVEAAIIDDSGNVL
PPNRMGNLAVKTPWPSMMRKIWNNPAKYEEYFRITGWYISGDSAYMDEDGYFWFQGRIDDVINTAGERVGPFEVESKLVE
HPAVAEAGVIGKPDPMRGEIIKAFIALREGFEPSDELKADIAKFVKEGLSAHASPREIEFKDKLPKTRSGKIMRRVLKAW
ELNLPTGDLSTIED
>lcl|NZ_CP026520.1_prot_WP_042232640.1_4407 [locus_tag=PC41400_RS22215] [protein=GNAT family N-acetyltransferase] [protein_id=WP_042232640.1] [location=4909845..4910483] [gbkey=CDS]
MDHKKIYHSHSFLPPGLDAEITVEGPVSPDHLRGLQMHADLDAFRKPKDQHEALIEIAELPEGRIIIARTGDWIVGYVTF
HYPDEIERWSQGNMPDLIELGAIEVANSYRDLGLGKKMIRLAFTDGQLENVITFTTEYYWHWDLESSGLSVWDYRKMMEN
LMKCVDMVWFATDDPEICSHPANCLMVRIGSKVPLASVEQFDRIRFQQRFMY
>lcl|NZ_CP026520.1_prot_WP_042232638.1_4408 [locus_tag=PC41400_RS22220] [protein=acetoin utilization protein AcuC] [protein_id=WP_042232638.1] [location=4910520..4911698] [gbkey=CDS]
MKNESVFVFNEHEATYKFNEEHPFNPKRLLLTIDLLRKAGALNDSEIVTPRVATDDELLRVHQPAYIDAVKALSELSPSL
DWVNSAAKYGLDTEETPYFAWMHDVTSMIVGGSITAADTVMSGRAKRAFHLGGGLHHALQSKGAGFCVYNDASAAIAHLT
EHYNARVLYIDTDVHHGDGVQWSFYTDPRVCTFSIHETGKYLFPGTGDVTERGEGEAFGTCVNVPVEPYTEDDSWLESFE
AVLVPLIERFKPDIIVSQHGCDAHALDPLAHVHCSMRIYKRMPELIRELADKWCDGRWVALGGGGYDIYRVVPRAWSLVW
LVMTGHPLLQELESNPLLPLPAAWVAEQQPGSPVRLPATWLDDVADWEPIPRREIITEKNKKTVEIALLYLK
>lcl|NZ_CP026520.1_prot_WP_042232635.1_4409 [locus_tag=PC41400_RS22225] [protein=5'-methylthioadenosine/adenosylhomocysteine nucleosidase] [protein_id=WP_042232635.1] [location=complement(4911858..4912556)] [gbkey=CDS]
MINKIGLIGAMNEEIELLVGGMSGVTETNKAGITYREGQFEGKSVVVCKSGVGKVNAAVCTQILIDSFGVDAVLFTGVAG
ALDPELNIGDIVISTTCMQHDMDVTPLGFPRGVIPYEEVSVFKADPQLVKLADEASRELFQGRTKQGLVLSGDQFVASRD
KVAELHQELGGTCTEMEGAAVAQVCSMNKVPFVVIRSMSDKADGSAHVNFAEFTKQASENSHRIIEHMVRSL
>lcl|NZ_CP026520.1_prot_WP_231392007.1_4410 [gene=ccpA] [locus_tag=PC41400_RS22230] [protein=catabolite control protein A] [protein_id=WP_231392007.1] [location=complement(4912596..4913600)] [gbkey=CDS]
MTVTIYDVAREAGVSMATVSRVVNNNPNVKPQTRKKVFEAIERLGYRPNAVARGLASKKTTTVGVVIPDISNSIFAEVAR
GIEDIANMYHYNIILSNADKKKEKEIRVINTLLEKQVDGLLFMGGVVTEDHINAFRTSSVPVVLCGTADEQNVMPSVDID
HEKAAYDAVQLLIESGHRTIAMISGPLQDPANGFARYQGYKKALEHAGIEVNEDYVRLGNYRYESGIDATDYFLKLPTRP
TAIFSATDEMAIGAIHKIQDEGLSVPKDISVMSIDNIRMASMVRPQLTTVAQPMYDIGAVSMRLLTKLMNKETMELTQVV
LPHEIIKRSSVAPR
>lcl|NZ_CP026520.1_prot_WP_241688716.1_4411 [locus_tag=PC41400_RS22235] [protein=hypothetical protein] [protein_id=WP_241688716.1] [location=complement(4913856..4914305)] [gbkey=CDS]
MMQKGYCCGQPIHLCLRTVVFKQKVEIENVPIYSCEACGKSVVYSSVKPKVTELIDTLGPKPEKQILRFEDVSEIACLIV
RLACSSEDGYGFKQAVEERVNELLDLLLLATSVGDMGWAGEIRSRLAQISEHMLSDDRLQEAFADRTCG
>lcl|NZ_CP026520.1_prot_WP_042232631.1_4412 [locus_tag=PC41400_RS22240] [protein=type 1 glutamine amidotransferase domain-containing protein] [protein_id=WP_042232631.1] [location=complement(4914471..4915016)] [gbkey=CDS]
MTVQDLKGQRILAFVDEEFEDLEMWYPVLRLREAGAVVDIAGPKAREIYHGKYGVPITTDVAFGDVSAADYDGLYVPGGW
APDKLRRYADVLRFTREIHEAQKPIAQICHAGWVLISAKIVEGYTMTSTPGIRDDLENAGAIWVDEEAVVDRNIVSGRRP
PDLPAFMKAFIEVLIQKKATN
>lcl|NZ_CP026520.1_prot_WP_042232629.1_4413 [gene=ptsP] [locus_tag=PC41400_RS22245] [protein=phosphoenolpyruvate--protein phosphotransferase] [protein_id=WP_042232629.1] [location=complement(4915048..4916778)] [gbkey=CDS]
MFKGIGASSGIAMGPAYVIPAWDWEFPDKMVDATDLSFEFERLYDGIRSSKDELEHIKQEIRDVLGQEQSYIFDAHLAIL
EDPVFMNEIQGIISRQYKAAEVAVKEVIDKFVEMFDVLDDDYMKERAMDIKDVGNRLLKHLLGSGDDTPPPTDHSYVLVA
KELTPSQLVHLDPARVLGIVTMMGGLTSHTSIMSRAMGIPFVLGLEGKLIRPIQNGDLMIVDGEEGIVYVNPDSGTVELY
ESRKTDWLMHRERLQEIAHVPSMTQDRKQVRLAANISSIKEIDQVLKNGAVGVGLFRSEFLYMDRDSLPDEEEQTEVYKQ
AAVKLDGRPLIIRTLDIGGDKKLDYLPLQVEENPALGCRAIRISLDRRELFKTQLRAILRASHFGQVKLMYPMISSLGEL
REANALLAEAKQELKSRGKPFNPDIEVGLTIEVPGAALIADVLAKEVDFFSIGTNDLIQFVLAVDRMNDSIAHLYNPYHP
AVIRLLKLTIDAAKAAGIPVAVCGEFAGDIRALPLWLGLGIEELSMSVQLILPIKHRLLTSNHAECERLLQEVLQCGTSE
EIEQVLRQADSRSSAS
>lcl|NZ_CP026520.1_prot_WP_042232626.1_4414 [locus_tag=PC41400_RS22250] [protein=glucose PTS transporter subunit IIA] [protein_id=WP_042232626.1] [location=complement(4916771..4918663)] [gbkey=CDS]
MNWMGNLQQFGRSLMLPMITLPVAAVLIRLGILPWDSLGAAWLGDMMLLAGHTIFTYLPLVFAVGVALGLTENAGIAGLS
ALMSYFLFTVLTQHQLGETFQLGVAGGILIGLLSAVAYHRCKRIQLPEYVQFFGGPRMVPLIMGVVTLVLSILMIKLGPV
LENGMEQLTAWVLSLGGFGAFLYGIAHRLLVPSGLHYILNNFFWFQLGAYKEPNGDMVYGDLPRFFAGDPTAGAYMAGLY
PIMMFALPAIAFAIIHEAREDLKPHVRVTFMTAALTSFLTGVTEPIEFAFLFVAPYLFVIHAILSGFAMWIAYALNISHG
FSYSAGAIDFVINEHLSRNGWMLIPIGLVFGAVYYVMFRYAIRRYRIPTPGREEGSSLEEWAGDIPYRAPLIVQALGGKG
NIKKIEACITRLRLTLESDRLMDIAALKHLGAAGVIRLGGGNVQVVFGTFSELIREEIMKLLRKDIQLVLFSSPMAGRMI
PLEDVPDRIFAGKIVGNGVAFIPDKGELLSPVAGKIMHVYPTMHALGIQTEEGLEVLLHIGIDTSTLTGNFFEAQVRPGD
EVEAGQLLIRFNYSQLKKHAKSLATPMVITNSDRVKSWSFAPYKSVKKGQASVMSVVLKKPEQSGGTSNV
>lcl|NZ_CP026520.1_prot_WP_042232624.1_4415 [locus_tag=PC41400_RS22255] [protein=hypothetical protein] [protein_id=WP_042232624.1] [location=complement(4918799..4919551)] [gbkey=CDS]
MYNQYQNQAGIGSQSQYQNKFQPTGYVQSSYGQNQYQNSQQYGSYQNPQSFHTANYRGNQPGHDSYLRSDSQNPAQQQGF
GQASQGLTQSYGSQAYGNQGNQFYTSQQQSYGQFAAPQSYHTANYRGDQPGHDNYLRSDSESPSQQGYGQASYGLSQSVV
GQGYGSQNYGSQSYGQSYGQGYGTQSYSQSASPQSYHTASYRGDQPAHDSYLTSDSQTPAQQQIGGYSSSVGSAQSGRFN
TGIPYTTGTF
>lcl|NZ_CP026520.1_prot_WP_042232622.1_4416 [locus_tag=PC41400_RS22260] [protein=CoA-binding protein] [protein_id=WP_042232622.1] [location=complement(4919747..4920172)] [gbkey=CDS]
MPFENPSRDEIKTLLASAGPIAVVGLSDNPERVSHMVSAAMKSRGYEIIPVNPNADVILGGKSYASLSDIPGNVDIVNVF
RRSDQVVPIAEEAVRKGAKVLWLQQGVYNEEAAEIAQRGGLKVVMDRCIKVEDAILNPRGN
>lcl|NZ_CP026520.1_prot_WP_042232620.1_4417 [locus_tag=PC41400_RS22265] [protein=C39 family peptidase] [protein_id=WP_042232620.1] [location=complement(4920225..4921019)] [gbkey=CDS]
MKVTFGFFLILGLLFSGGFMSVMLYAKISGQDVQTVMAMDAKLEKQPPPEPEQPPAAEQVQEQPKTKAMIDAPVIAQKPE
LPSGCEITSIAMMLHYKGIKLDKMQLYDEMPKDPTPIVWNKDGTIASWGHPNIGYVGDGTGKSKGFGIYHKALFPLLKSY
IPSAVDITGESFDVYEKQIANGRPVVVWTTIGFTMPTRWSEWNTSIGPIRATFSEHAVLMVGYDETSVYVNDPWTGSKNV
RLDKKLFIDIWEAMGKQGLSYTAD
>lcl|NZ_CP026520.1_prot_WP_042232830.1_4418 [gene=aroA] [locus_tag=PC41400_RS22270] [protein=3-phosphoshikimate 1-carboxyvinyltransferase] [protein_id=WP_042232830.1] [location=complement(4921054..4922352)] [gbkey=CDS]
MKAIVKATPNLQGEIQALSSKNYTTRYLLLAALADGTSTIHYPAHSEDSDAMRRCIRDLGAVIEEDEHKMVIKGFGKNPL
PIKELNVGNAGAVLRFLMSTAALCQSEVTFINQYPESLGKRPHMDLIDTLEQMNVQVEHNEGRLPITIKGGQPRGGKITV
SGSVSSQFLSSLLFLTPLLQEDSEIEVLHDLKSKVIVGQTLEVLEQAGIRVEASEDLMHYRIPGNQSYRAQEYVVQGDYP
GSAAILAAAAVTNSDVTVHRLDENSRQGEKACVDVLRAMGVSLTHKDGIVRVQGTGKLKAGEFDGDHFTDAVLAMVAAAV
FAEGTSRFYNVENLRYKECDRITDFLNELRKAGADVEERQSEIIVHGRPEGLEGGVEINAHYDHRVIMALSVAGLRSKQG
LVIRDAHHVAKSYPQFFDHMISLGAHIELTNE
>lcl|NZ_CP026520.1_prot_WP_042232618.1_4419 [locus_tag=PC41400_RS22275] [protein=CrcB family protein] [protein_id=WP_042232618.1] [location=complement(4922410..4922769)] [gbkey=CDS]
MNAGWMNVLLVGAGGFVGSIARYKLAGWVQSRTASPFPFGTLAVNLTGCFFLGWLFGHVTNAALLIGLGTGFTGAFTTFS
TLKLDSWKLRKKKLFNLDMIYLAVSYLAGIGLAFAGFYI
>lcl|NZ_CP026520.1_prot_WP_042232616.1_4420 [gene=crcB] [locus_tag=PC41400_RS22280] [protein=fluoride efflux transporter CrcB] [protein_id=WP_042232616.1] [location=complement(4922766..4923143)] [gbkey=CDS]
MKSWIIGTAGVFGALSRYGLSVLWNEGTGSFPWGTLACNMIGCMLLGYFSEALLPKWPQELRTAVMTGFIGAFTTFSTFS
METVRMLQDHRVPLALLYIGVSLLGGLLAVSLGTAAAHRRKEKAI
>lcl|NZ_CP026520.1_prot_WP_009672581.1_4421 [locus_tag=PC41400_RS22285] [protein=rhodanese-like domain-containing protein] [protein_id=WP_009672581.1] [location=4923275..4923568] [gbkey=CDS]
MNKIQPSELKARLAAGEPLCVIDVREPEEVAAGQIAGAKSIPLMEIHTRLHEIPKEGEVILVCRSGNRSGKAYDFLESQG
FANLKNMEGGMLAWEQL
>lcl|NZ_CP026520.1_prot_WP_042232613.1_4422 [locus_tag=PC41400_RS22290] [protein=shikimate kinase] [protein_id=WP_042232613.1] [location=complement(4923595..4924143)] [gbkey=CDS]
MSTQPRSNIVLIGFMGTGKTTVGLKLADRLGWPMIDTDEEIVKKAGMSIPELFATHGEAHFRTLEREVVRETLAGSHRVI
STGGGAVLAEENRACMSERGFVAALSASLETIVERVKNDSNRPLLQGSGLEEKVRTMLQERQDAYRFADLTIDTSPLAAD
DIAEHIFNAWTKERLKREELSG
>lcl|NZ_CP026520.1_prot_WP_042232611.1_4423 [locus_tag=PC41400_RS22295] [protein=hypothetical protein] [protein_id=WP_042232611.1] [location=4924327..4924671] [gbkey=CDS]
MMPHPRIPDPSRNTSGRLSQMLQPSYMLCRDDVVWALEFIRKKMAEQDSRLKELTQPRLLKNFESFAEVSMMLVHRRSAF
DQEADRIKSCLKEASYGLFENEPPSERDSSPAND
>lcl|NZ_CP026520.1_prot_WP_042232609.1_4424 [gene=gndA] [locus_tag=PC41400_RS22300] [protein=NADP-dependent phosphogluconate dehydrogenase] [protein_id=WP_042232609.1] [location=complement(4924762..4926174)] [gbkey=CDS]
MSKQQIGVIGLAVMGKNLALNIESRGFTVSVFNRSPEKTKELLEEATGRNLVGTFSMEEFVQSLETPRKILIMVKAGQAT
DATIEQLVPLLSEGDILIDGGNAYFPDTQRRNKELEARGIRFIGTGVSGGEEGALKGPSIMPGGQKDAYKLVEPILTAIS
AKVNGDPCCTYIGPDGAGHYVKMVHNGIEYGDMQLICEAYQLLKDVLNVSTEELHEIFTEWNKGELDSYLIEITRDIFTK
YDEETGKPMVDVILDSAGQKGTGKWTSQSSLDLGVPLSIITESVFSRFLSAMKEERVAASKVLNGPAVKPFEGNREEFIE
AVRKALYASKIASYAQGFAQMRAASEEYNWNLSYGEIAMIFRGGCIIRARFLQNIKDAYDRDPELRNLLLDDYFKNIVEN
YQDAWRNVVSVAVTRGIPAPAFASALAYYDSYRTERLPANLLQAQRDYFGAHTFQRVDKEGTFHYNWLDS
>lcl|NZ_CP026520.1_prot_WP_042232607.1_4425 [locus_tag=PC41400_RS22305] [protein=hypothetical protein] [protein_id=WP_042232607.1] [location=complement(4926739..4927722)] [gbkey=CDS]
MKTAKLAIFFVLIYVPFAWMNKFDAKAQFQMLFLETKYDAAVNAAVQDGAAALSVNEKQGEEVRYESSKKVRVNRKEAVD
SFYRTFYSNFGVSENAAGQQVLDSYLSALLIVGYDGCYSYTIESYTGEDGFERSRRLEGPKRPYVYQDPTGEIIAFTLDE
YVTVKNAAGTGWIEGFRSEIQAKTQTALLKPAARFDEVRRGAIINTIQRELENAIREHNIQTRRLGAAYTFTLPLLSGEE
WSNTINDVGMLAFVQGIPIGYRTYNNYALGGARIVKRPVFYGYRQNNLPYFTSSRCPRPQEILETFTSPKAAAAAGYYLK
ECVNGRT
>lcl|NZ_CP026520.1_prot_WP_042232605.1_4426 [locus_tag=PC41400_RS22310] [protein=hypothetical protein] [protein_id=WP_042232605.1] [location=complement(4927709..4928248)] [gbkey=CDS]
MDTLSKAFVLILAVLLLYIYPLSDSAAREDDLSRVIAFETMTRFVDNVRDKGRLTPAMYEQFQKELTTSGQEFDVNMEHK
HKRYVPVYDDQGRFAEKYTVEDEAFYNAQILSVLFPENSLPPDHADRTYKLGAGDYFTVTVHSLSRTPAVVWQDALNGTN
TPGGQLYMPYGGMVRNEDS
>lcl|NZ_CP026520.1_prot_WP_042232603.1_4427 [locus_tag=PC41400_RS22315] [protein=hypothetical protein] [protein_id=WP_042232603.1] [location=complement(4928278..4928664)] [gbkey=CDS]
MNHATRSMLFAGVSVCIFVSALVTGMMLLQNVRGAIADTYLSRKLLDYNQQPANAKLGAVSVKGSDLIMSIYLNAGEPYD
IVVNGKRYPGGRQKEETDPAGIDPAAAYRYYVVRDTGGTLKRIVYTTP
>lcl|NZ_CP026520.1_prot_WP_009672600.1_4428 [locus_tag=PC41400_RS22320] [protein=hypothetical protein] [protein_id=WP_009672600.1] [location=complement(4928673..4929152)] [gbkey=CDS]
MGNAQKALIMAAGFFLAIALITIAVVMFISAQDATKAAQNNFSDIQTELSQTAFTVYDNTVVSGSQVVNALRKFADKEQF
GIQVQTGKNPVGSWYTNKIDTTSPSSSNYGTVIASISRDAVANATLESHMDYVNPSGKFKAHIIKDKSNVIRAIEFMQQ
>lcl|NZ_CP026520.1_prot_WP_042232600.1_4429 [locus_tag=PC41400_RS22325] [protein=hypothetical protein] [protein_id=WP_042232600.1] [location=complement(4929174..4931255)] [gbkey=CDS]
MPLRDWLLAIALFSGGTACVIWFLYVYAIKRRAKLSGYGEDELQKLLSPLKGNGRKQIERLYGKCYEIMSNIPGISRYTQ
RIRRRIAPLYSYDESMIRMETGRITLLAFLIGWLTAGFVFLFYPDVSYMLMILLGSLVLHGILIDTLVHRAEDRLLRQMT
EFLSDVRHFYHQHGMVDEAVYDAAQIAPRQLSLHAEAVYGTLNSDHPERSLEIYYEHAPNRFLKSFAGVSFLVREFGDKL
LPHGSMYLHALSKITSEIRLEILRKEKLHYLLKGLSVIAISPIMAAKPIEAWARHNFPAMDEFYTGAAGFAVKTGIFVLI
IVCFVLLRKLQDTAEETGSRRTGRKPPEQKWLGNRLVNAVTQRVVPAVHSMERGRLQKLLREANSPLLLEWLTLRRLLTG
LACFLLVLGMFLIMQRLTLQQVYFSPTRNQAMFGSLSPQEKRKALETAAFDRNVIQKLERSGRELREMIAEEMRQQGFQP
DPTVLNTAGERIKAKLDSIAGAYLSWRELLFCFISGWAGYHSPVWLLHMQKRMRRMEMKNEVDQFYTLLTILAHFERMSV
EHILEWLERFAVLFQSPLSACLLDYEAGADEALEKLKEEAAYTPFVRLVERLQAAVNHIPVKQAFDDLEAEQQYHFERRK
QDYERMIETKAGWGRWIGFAPMYALVFLYLVIPLVYMSMQQMNLYYQQIGKIS
>lcl|NZ_CP026520.1_prot_WP_042232597.1_4430 [locus_tag=PC41400_RS22330] [protein=ATPase, T2SS/T4P/T4SS family] [protein_id=WP_042232597.1] [location=complement(4931242..4933146)] [gbkey=CDS]
MPTDRLNLALILIVAVFAAVWFFAKTGGWRVRTSASQTAKNYSIEAMTAFIKEAFAEITAGGGSELRLSEEEYRRSRNKR
LQLKRALKGCGHGDLKDKRYVKSVMMDLLTGVNGLGGTTIHHSLPFHKPEQLTAQDRFDILLHLYRKKYGYDGLHELIRK
HKLDAVKHVIEDGRTPSYIVTAEEIAHVFRKEAGLLSFQDKLEIVVQRIYQHYKGFGVIDAIREMRIDGVSGGVSGALPD
SAGYRGAPLAEREADGNHRYKKESTSYDSVWIFYRGKPIHFSFLSFGSSEELRRVCQNIYKYNYPGQLSETNGYKVNDMK
DGSRVVVVRPPFSESWAFFVRKFDLPNATLDQLVHGGNAGLAVELIRHLIKGCRITAVTGAQGSGKTTLLMAMVQSISAT
LTLRVQEMAYELHLRNLYPQRNILSFRETDHITGQQGLDVQKKTDGAVHILGEVATDTVAAWMIQMAQVASLFTLFTHHA
KTFASLIDSLRNSLLKTGMFQDEVMAEKQVIRVIHFDIHLRKSADGTRFIERITECVPLDSDLYPSEKAGMTDKDKVELP
GFLDDASEFFRRTTDRKLYEQRNIVEYRDGAYRVCRPISLRNTEEMLDHMTPADAQAFERFLETHWGGDVIAAS
>lcl|NZ_CP026520.1_prot_WP_042232595.1_4431 [locus_tag=PC41400_RS22335] [protein=hypothetical protein] [protein_id=WP_042232595.1] [location=complement(4933153..4933983)] [gbkey=CDS]
MSQVAFWAPVRGHGAATSNVLAVSTMIALDYYARIMITHTKRTRTALENAFGRPSSDGNLLSFSEYGLDALERLLQSDKM
MPETIQDYTKPIVKDRLDLLPGSRKPLADPDCFREETLPYIVNSASRYYDLTFVDVGDGYKEDSTEVILRNSDLIVVNLS
QNLELLERYFNKELWTESLQNKPLLLVLGDYDGNSRFNAVNLARKFKWRQPIYTVPHCTGYLDACCDKNVLEFFLRSRNA
GPGHSSYAFITEIRKLTKALLERLGLDSKLMQERGA
>lcl|NZ_CP026520.1_prot_WP_042232593.1_4432 [locus_tag=PC41400_RS22340] [protein=hypothetical protein] [protein_id=WP_042232593.1] [location=complement(4933980..4934687)] [gbkey=CDS]
MKLLCVGAKEHSDLLLGIGRLAAVGGRRVLLIDGTSSRRLSCLVAQGLEPELVQYDEIDVACHFASLDAALEHTAASGSV
LEEYDIVMVDTDRADFPGAAGAYPFEGLFLTTAYDKYTIRRTAELICSLNDVCQVNTLTSTVIIRDAVECGIDEAYVDKV
LAPLPFITPDEYHKLPFDEVDLAVRIDSEYQGKVGIRRMSRAYRNVLLAVANEITGEDRAVLKRIIKQAMRRKTG
>lcl|NZ_CP026520.1_prot_WP_042232591.1_4433 [locus_tag=PC41400_RS22345] [protein=SAF domain-containing protein] [protein_id=WP_042232591.1] [location=complement(4934719..4935798)] [gbkey=CDS]
MSIIRQRTKHLIYSGLIGALTAGLLASGGFVYTGLQHRKKLTELEYRYQADLSEAKSSSANTERRKVVVLTRDIEAGTRL
GDEDIKYLEMNAADAPVGSDLDLRQVAGRTFKLAGVKNTPVLASMLYEEGMLAKDVRQEEFNVIGLPSLLKKDDFVDVRI
SFPTGQDYIVLAKKRVDNVSEGTVWLKLSEREILDMSSAVVDAYLHGAKLYSLPYVDPQMQDKAEPTYPVNAKVLELIRT
DPNVLETAKLELMKSMRQLLDKDLQNMDDASKQKINAGISSGRTGGNPAGIGAVQGEEPPSGFGPSTRPPAKPEAKPGGY
SPGTSGSPEKMPTENAGPETPPSEIRERQEDIFREVVQP
>lcl|NZ_CP026520.1_prot_WP_042232589.1_4434 [locus_tag=PC41400_RS22350] [protein=protein kinase] [protein_id=WP_042232589.1] [location=complement(4935815..4937314)] [gbkey=CDS]
MKPDPGRLYEGDILGSRYRIVSTVGRGGMGTVYLAEDLKLKGKRWAVKEILQQVQDYQRFVDEAEMLIRLHHAHLPGIVD
YFPPDDEGFSYLVMDYIAGETLAERFSRTGGQVGVAAAVNYALQLCDLFHYLHDELPVPMIYRDLKPSNIMIDGQEQVRL
IDFGIARHFKSGGQADTVQIGTVGFAAPEQFQQLQTDRRTDLYSLGAVLYYLLSGGKYPHSFGVPLAGLNEEVPHSLSVI
IEKLLEHDPDHRYPNARELREALSGVYAALRGGEGPSARSGGFLGPVVIAVCGTGRGVGCTHTAVLLAHFLARKRRTVVL
AEANASGHFSRIEAVYEGSDGPLCGSAAFDLRGVRYAKAGDGLDMIGTLSGSYDFVVLDLGSYDETEWFGEFMRAAVQVV
VGTGCEWKRRDIAAFFRSHPVPEPAKRSLFVPLASAQMLRDIRRMLPDVRLIGLPAHADPFCHQQETAELLDGWLGLPEA
RTQGAGPFMRRWFRARAPK
>lcl|NZ_CP026520.1_prot_WP_042232588.1_4435 [locus_tag=PC41400_RS22355] [protein=hypothetical protein] [protein_id=WP_042232588.1] [location=complement(4937512..4938108)] [gbkey=CDS]
MSKNHEVEYCNLELRFDRRLIRQFIKALIQEGYSLYWNENDHQFVISIRSGRKLVKLKFERIGERYKIVGNYSFKDEKLA
EMMEKLIGDTRGHAVVKRFKDRQILIENIMFGEIIRMVEISGIEHKVLFQKEPVVTFEEMLLAFRSRRIEERIPVLRLEL
DYELAVLNEAIQKGDVQAVVDSKEKLTELRQEMILLEM
>lcl|NZ_CP026520.1_prot_WP_156972851.1_4436 [locus_tag=PC41400_RS29950] [protein=hypothetical protein] [protein_id=WP_156972851.1] [location=4938325..4938486] [gbkey=CDS]
MIKPIAAYILALFSLGCKMAEPTASGTTAINPKIADGLSNKEIAIFYCKQARP
>lcl|NZ_CP026520.1_prot_WP_042232586.1_4437 [locus_tag=PC41400_RS22360] [protein=stalk domain-containing protein] [protein_id=WP_042232586.1] [location=complement(4938521..4939360)] [gbkey=CDS]
MHMPSKKMIAAAAGVLMSVSFAAGVYADDWIQKVTAYVRSDFKVVVNGTPANLSSPVLIYDNNSYLPLKEIASRLNANVN
WQDSNQTIYVNTRLYPQQPETGDVVYDEIILENPTAYIMKYLGGDYPVLVTLSKGMYYRQSDVQKMGMDTKGLRKAKEKY
TGEIYINETELRKVWKENPILTYGEEMPPIIAGEKDDFKIRAIRGYVKDMSNYKIGDTYYFHNPIFIEKLEEANTYRYLL
TENGQYYQVNLKLTEMQKDLYIVGSSSKQLLGKTAVDPY
>lcl|NZ_CP026520.1_prot_WP_042232584.1_4438 [locus_tag=PC41400_RS22365] [protein=DUF1054 domain-containing protein] [protein_id=WP_042232584.1] [location=4939517..4940167] [gbkey=CDS]
MSNSTTIRPAFQGFDSSDFETFAIEGLDERMKAIQERIQPKFKMLGETLCEDIALHAGTEMHLHIARHARRTVNPPKDTW
LAIAANKRGYKQHPHFQIGLFDDHVFLWFALIYEVPDKQAIAGRLLASQDELYRSIPEHYVLSFDHMQKRSVPAGDLTAE
AFGDALVRFRDVKKAEFLVGRHFDQQDPILQKGDAFTRMAKETFESLMPFYRTARG
>lcl|NZ_CP026520.1_prot_WP_042232582.1_4439 [locus_tag=PC41400_RS22370] [protein=hypothetical protein] [protein_id=WP_042232582.1] [location=4940321..4941046] [gbkey=CDS]
MFLQPSKKKWAGTLALAFLLGSVSLGARQAAAFGSPEELPDEAVLSLPTPSAGVPGRQDSHNPKDGSVSGLPILEEAASI
LGTTVASLKESLDKGKSLSDVAKDKGISEKELTAKLLTIRDSKIDQAVKEGRLDAERAAALKQKMAEHLKRMLQDKALLD
HHNPHQHGKMRHSLKPDQEKLAQAIGISKEELHRELRQGKSLQQIAESKGITKSQLISRIQALLAPDLEKYVEQRVPARE
S
>lcl|NZ_CP026520.1_prot_WP_042232580.1_4440 [locus_tag=PC41400_RS22375] [protein=right-handed parallel beta-helix repeat-containing protein] [protein_id=WP_042232580.1] [location=complement(4941148..4942200)] [gbkey=CDS]
MKTKLTPQRLLRKSLLSAALLGAAFAPAMYAAAPLESGALADQAAAGIEEQAAGSLRTVSCSSVSCLQNALKNALPGDRI
LLAPGTYKGTFSSAVNGTKANPITIESRDPANQALLSGTSAGGGYCLKITGDNWVIKNLKFTNAQKGIILDNSNDTLISG
VEVYGIGYEGVHFRDGSSRGILENSYIHDTGTVNKGFGEGVYVGSAEGASYNQTVNDTIIRGVRIGPNVTAEHIDIKERT
SGTLVENCIFNGTGISGENYADSFIDVKGNNAVIRNNTGWRNGNTVITDAFQLHQIVAGWGVNNDFSGNTVNLDQSSAYV
INASQGTEAKAHDNVRNPAGNMYKGNVTTY
>lcl|NZ_CP026520.1_prot_WP_241688717.1_4441 [locus_tag=PC41400_RS22380] [protein=MFS transporter] [protein_id=WP_241688717.1] [location=complement(4942514..4943905)] [gbkey=CDS]
MNLAGWKNKAGAAAPFVVQLLVIMFVVEFVKGALLLTILPVYMKTVLGATAFVVGWSLAAQYIGDNVFRTPVGWLIDKAG
YRLSMLIGILLSGSSVLIMATFTHHAWMVAACAMLGVGTAPLWPAVIAGTTEVAGDKAKGTIMSVVYMAWLSGTGLGPVV
INFFIHGSNYGPAYRLLLISMGVVLLIALFLPRKPLLREDGEKREESRHAPAGAAFPAGAAMEAGADPLGEAVTAAGGGT
AAAAAAVMPAAGKKPLGERVKLYFGEVRASLREMNVSVLLFPAMFIQTFALGVLTPILTLYAREVLHLSPGQYSMLLIVG
GGAAVALLVPVGKLVDRKGIKGLLTAGLIFSSVSLLALGFIRQLAAVYVVVGLLGIGYALIIPSWNALIASCVPKEKRGA
VWGFFLTIEGLGMIVGPIVSGKLWDAFGPSVPFTVSGSVLLLLLALQWRILAKSRGNTANRTG
>lcl|NZ_CP026520.1_prot_WP_042232576.1_4442 [locus_tag=PC41400_RS22385] [protein=aminotransferase class I/II-fold pyridoxal phosphate-dependent enzyme] [protein_id=WP_042232576.1] [location=4944091..4945560] [gbkey=CDS]
MDQHRTPLFSALRTHAEKNPIQFHIPGHKKGVGMDPEYREFLGENALSIDLINIAPLDDLHQPTGVIEEAQKLAAQAFGA
EHTFFSVQGTSGAIMTMIMSVAGQGDKIIVPRNVHKSVMAAIIFAGIKPVFVSPARDANLGIDHGITTKSVRRALEKHPD
AKGVMVINPTYYGVCTHLKEIVELVHSYDIPVLVDEAHGVLIHFHDKLPMSAMEAGADMAATSVHKLGGSLTQSSVLNVQ
GSRVNPKRIQTIMSMMTTTSTSYILLASLDTARRHLALNGHELAERTIALAEYAREEINKIPGLHSFGEEILGTEATYAY
DATKVTIHIRHLGLTGYDVENWLRDRYNIEVEMSDMYNILCLVTPGDTSDNIDVLLQALRELSAEHYTGNAEDDLIVIIP
EIPQLSLIPRDAFYGDTEVVPLKESAGRIIAEFIYVYPPGIPILLPGEVISQENIDYIVEHVEVGLPVKGPEDRSCQFVK
VIVETQPIF
>lcl|NZ_CP026520.1_prot_WP_042232574.1_4443 [locus_tag=PC41400_RS22390] [protein=DUF1885 family protein] [protein_id=WP_042232574.1] [location=4945644..4946063] [gbkey=CDS]
MSQSAFIRFVTGSTMTSPTLDELKSQLEHYREQLSRTGAQLSWEYDSAGFPYTVETKPEGEGKWFYLKGTTPRYRSIVLG
VGSETQEDSLQHYVQIVLPDTATHGDKSKANELGKYLARHWKARLDLFNGRTMYFNPRK
>lcl|NZ_CP026520.1_prot_WP_042232572.1_4444 [locus_tag=PC41400_RS22395] [protein=DUF1292 domain-containing protein] [protein_id=WP_042232572.1] [location=complement(4946166..4946483)] [gbkey=CDS]
MSDHKHEHDQDCGCGHDHDHEHEESVFIVTDEEGKEHEMVMVYTFDSQDQLYAVLLDRNDPEADGVIFRIEEENDDAYLV
NIEDEAEWERVVAVYNQIIAEENRE
>lcl|NZ_CP026520.1_prot_WP_241688718.1_4445 [locus_tag=PC41400_RS22400] [protein=alpha/beta fold hydrolase] [protein_id=WP_241688718.1] [location=complement(4946638..4947603)] [gbkey=CDS]
MNALIWIVWGVVLLLIAGLLASVGIAAYVGWQLTHPKRQPVDDSPANYGLVFEEVEIASRTGDILLSGWHLPPPDTAVRQ
NPMTVIMSHGYAGNRLERGLPSLALARDLITAGYRVIMFDFRNSGMSQGSMTSVGYYEKHDLLGVIDWVTETYPHEAVSL
IGFSMGATTSLLAAAEHEKVAAVVADSPFHHLKKYLKSNLPVWSGLPNIPFTPLILLLFPPLLNIDPEGVDAVKAVEAIY
PRPVLFIHSSDDRAIPQASSEEMWRRHPDVFDFWRTAGAPHVGSYHLESAEYARRVIAFLEKTVTEETVHPREVGISCNS
H
>lcl|NZ_CP026520.1_prot_WP_009671838.1_4446 [locus_tag=PC41400_RS22405] [protein=aspartyl-phosphate phosphatase Spo0E family protein] [protein_id=WP_009671838.1] [location=4947678..4947851] [gbkey=CDS]
MLKTMNRSLDLEIEYLKSVLTYMAAQYKYELNHPRVVEVSQQLDGLIVEQMKKRAAS
>lcl|NZ_CP026520.1_prot_WP_042232568.1_4447 [locus_tag=PC41400_RS22410] [protein=cytochrome c biogenesis protein CcdC] [protein_id=WP_042232568.1] [location=complement(4947884..4948351)] [gbkey=CDS]
MGSNLHAISLFGTLGMALLVIAIRMKASKRPINAMKILMPPIGMSTGFLMFVAPLTRIPVTWGLIAFAVGAVVFAYPLIR
TSGFSKQGDHVYLHRSKAFLIILVVLLAIRLFLHSYVEEYVTVYQTASVFFILAFGMLLPWRVAMYVQYRKLMNS
>lcl|NZ_CP026520.1_prot_WP_042232565.1_4448 [locus_tag=PC41400_RS22415] [protein=MBL fold metallo-hydrolase] [protein_id=WP_042232565.1] [location=4948552..4949295] [gbkey=CDS]
MSLQLQMIGTGSAFAKKYYNNNALWTCSGFTLLVDAGNTAPRALHELNMKPDELDAVLITHLHSDHVGGLEELALQMRYI
YGRKLRLLLPAALEAPVWEHTLKGGMENTGENLNRLADYFDVELLEEGRPSEIHTGFRIEIIRTPHIAGKPSFSLFVNDT
IFYSADMIFQPELILNEVVGQRDCRHIFHDCQLEGPGVVHAALEELLTLPESVQSRIRLMHYGDNMPAFVGKTGKMSFIK
QHQIYTL
>lcl|NZ_CP026520.1_prot_WP_042232564.1_4449 [locus_tag=PC41400_RS22420] [protein=ATP-binding protein] [protein_id=WP_042232564.1] [location=4949462..4951093] [gbkey=CDS]
MIYKWFVIFYASVISIASWKLGLSVSGWLPLSLLFLAAAVLGLKGSSLTFSAVKLVLLTLFHWSSQTDWCFTLYLLLLAS
EVYESKSRKTVWMLVTLNLVTFAVIALITAPAFSVYPPHWMVLLYLSNFSIFMGLALTVFNTFRQRDKLAGERNQLMTHD
SLTGLPNFEACHAMLHNLVDKESRLLFILIDCTDLKAMNKTQGFQAGNQILKQIAVLLKIKFSQAAFISRYGGDEFAVAV
TYEDKDAMRAYVDQQLGSELPKLTGIQITYGIAVLPDDASSKDDLVLVAENSLFSMKREMWLKREEHMLRSEKLRIVGEL
ASGMAHEIRNPLTTIKGFLQISKSNGYNIENWYPLIMDEISRMSELTAEFLQFSKPHAVQFRVLSMQVCIQRVISLTLSE
MSRLGHEIVYDPPEKPLNIWMDPDKMVQLLINLFKNAYESMEDNKKMAIRLYERDHHAVLEIEDQGSGIPAEQLDQIFNP
FFTTKDSGTGLGLSICHKIVQDHNGTIEVESFIGRGTTFIMTFPLAEPESTPCYPLNRRKDAV
>lcl|NZ_CP026520.1_prot_WP_042232556.1_4450 [locus_tag=PC41400_RS22425] [protein=GNAT family N-acetyltransferase] [protein_id=WP_042232556.1] [location=complement(4951097..4951534)] [gbkey=CDS]
MEIIRVTTDEQLKECLSVRKEVFVDEQNVPLELEIDELDASHQSGHHFLIREGGKAIAAARWKPYEENRAKLQRIAVLKS
SRGTGLGSVLVQAMEAQAKELGYTGAVLDAQCQAEPFYKKIGYETIPGEPFDDAGILHVRMVKPL
>lcl|NZ_CP026520.1_prot_WP_009671848.1_4451 [locus_tag=PC41400_RS22430] [protein=CBS domain-containing protein] [protein_id=WP_009671848.1] [location=4951755..4952168] [gbkey=CDS]
MNIAFFLLPKLEVITTTIDATLRQTLERMEYHRYTAVPILTEDGKYMGTVTEGDLLWFLKNSPGITFDNAHRHTLEEVPL
RMRNKPVHIHANMEDLIDLAKVQNFVPVVDDAGNFIGIVRRSDIIEYCAKQMFQAPV
>lcl|NZ_CP026520.1_prot_WP_042232554.1_4452 [locus_tag=PC41400_RS22435] [protein=hypothetical protein] [protein_id=WP_042232554.1] [location=complement(4952320..4952502)] [gbkey=CDS]
MRNIDIRVMSKQLGFGTESLESWLTKERQQEPENQPAYRLESLRPCYENRPSDEFVSLSL
>lcl|NZ_CP026520.1_prot_WP_042232552.1_4453 [locus_tag=PC41400_RS22440] [protein=hypothetical protein] [protein_id=WP_042232552.1] [location=complement(4952692..4953333)] [gbkey=CDS]
MIKTQELKQQMDDLSKQNDMLLVEFEQLKKMIMNGGSQNSGDSSGGSDSSGGGQYETGQSTNDGQQQSQNQNQSQGQAGT
GGSQSTRRSASSGNRQQSQGGQNQQDSSSGSGNQQAQGSSQQQNGNSGNNGSSGNNGSPAAVSEMADDLYVIKNMVEQLE
RKTSQYVSEQTNQSLSEKDVVNLVLYLMNGMIDWTSEFVGKNQSSSGSSGPMQ
>lcl|NZ_CP026520.1_prot_WP_042232550.1_4454 [locus_tag=PC41400_RS22445] [protein=superoxide dismutase family protein] [protein_id=WP_042232550.1] [location=complement(4953363..4954004)] [gbkey=CDS]
MRKPILSLLLSAAAVVAMTGCGSGNNKNAGPNSTVTPEALPTAATAPSASPGATPSASPSSSSGVEIELKNGEGSVIGTA
RFTQENKGVRIQVQATKLAPGEHGLHVHESGVCAGPDFKTAGGHFNPNSNLHGMENPQGPHVGDLPNLKAEANGEAKLDT
VSEAFTLEKDKPNSLLKQGGTSLVIHEKQDDMKSDPAGNSGNRIACGTISEGK
>lcl|NZ_CP026520.1_prot_WP_042232549.1_4455 [gene=glpK] [locus_tag=PC41400_RS22450] [protein=glycerol kinase GlpK] [protein_id=WP_042232549.1] [location=complement(4954254..4955744)] [gbkey=CDS]
MSSYIMSIDQGTTSSRAILFNHEGQIVHTAQQEFPQYFPQPGWVEHNANEIWASVLAVVATCLIESGVKPEQVAGIGITN
QRETAVVWDKRTGEPVYNAIVWQSRQTQQICDELKQRGLEQKFRDKTGLLLDAYFSGTKVKWILDHVEGAREKAERGELL
FGTIDTWLIWRLTVGEAHVTDYTNASRTLMFNIHELRWDEELLELLTIPAGMLPEVRSSSEVYAHTSESHFFGAKVPVAG
VAGDQQAALFGQACFDKGMAKNTYGTGCFMLMNTGEQAVTSSHGLLTTIAWGLDGKVQYALEGSIFVAGSAIQWLRDGLR
MIKTAKDTETYAARVESTDGVYVVPAFVGLGTPYWDSDVKGAVFGLTRGTTKEHFIRATLEALAYQTKDVLAAMEADAHT
KLKKLRVDGGAVSNDFLLQFQSDMLGVPVERPLINETTALGAAYLAGLAVGYWKDCAEIAAQWSVGGTFEPEMPQQRSEQ
LYSGWQKAVHAAMAFK
>lcl|NZ_CP026520.1_prot_WP_042232547.1_4456 [locus_tag=PC41400_RS22455] [protein=VOC family protein] [protein_id=WP_042232547.1] [location=complement(4956025..4956411)] [gbkey=CDS]
MAVTKLEHVGVMVADMEKSISFYESVVGLKLKDTVTNGAMKLAFLGFDVKGETEVELIEGYGGSLTEEGRVHHLAFTVDD
IEAEFERVRKTDVVMIDTEITTLPNGSRYFFFHGPDGEWVEFFQSTRA
>lcl|NZ_CP026520.1_prot_WP_042232545.1_4457 [locus_tag=PC41400_RS22460] [protein=hypothetical protein] [protein_id=WP_042232545.1] [location=complement(4956617..4956853)] [gbkey=CDS]
MIMIVWIIVIGFIVLRLATSAYNPPKNRKNKHHYSDSGSSGYTDSGWSFWGGGSDCDSGSGGGSDCSGGGDGGGGGGD
>lcl|NZ_CP026520.1_prot_WP_042232543.1_4458 [locus_tag=PC41400_RS22465] [protein=hypothetical protein] [protein_id=WP_042232543.1] [location=4957066..4957263] [gbkey=CDS]
MNKRSNDDIGIYNETSNTAGGINTPPDREDKAATDPVSDIVEEIMDNLSGEHPTDPSSDKPADKR
>lcl|NZ_CP026520.1_prot_WP_042232542.1_4459 [locus_tag=PC41400_RS22470] [protein=DUF4177 domain-containing protein] [protein_id=WP_042232542.1] [location=complement(4957366..4957566)] [gbkey=CDS]
MERWEYQTLKFFTKGFFVGGKLDLDEFDQDLNEMGAQGWELISCFDTSQHQGSSKEVICVFKRKLF
>lcl|NZ_CP026520.1_prot_WP_042232540.1_4460 [locus_tag=PC41400_RS22475] [protein=hypothetical protein] [protein_id=WP_042232540.1] [location=complement(4957734..4958129)] [gbkey=CDS]
MKRIISSRLAGNLLVLLNTGMFLVHVLILLKVIPSHFIWGGRINGESELVALESVSVVIQIVFIALIALKTGYLLPGRFK
RTARIGLWLLFVFMVLNTAGNLASISGPEKLFMTPLTVVLAFLSLRLAVEK
>lcl|NZ_CP026520.1_prot_WP_042232537.1_4461 [locus_tag=PC41400_RS22490] [protein=hypothetical protein] [protein_id=WP_042232537.1] [location=complement(4958826..4960130)] [gbkey=CDS]
MIDLVKIITIAWGTAWLIAAIKQLSQGTKSTVWFIMIIWYTFFIVPLILDLSLGSPHYGGYKGLRIAANDSNANLIYCLY
MCVVPLVFYLFGRAKKVHIEQSSTKPHKLLIAVLQILMISPIVVACFSPHPELYLKYGSIIAEMENFPEAEQFNKIVSLC
TLVSVAAVGILLMLRPRLTVFSFVPYLPWLFLAVWLNGKRAVFIMALVLIVYAYYQKGKIKGLKLLGALILSVSLMAGFS
NFYQNELRYEQMGITDSDDIYKNVRVDFGRDDRTKMTIYAELHPQTMKILDYRGQSLLFDLTFFVPRSMWPDKPWPYAVY
FTSAMFMMPPDYLGFGMTTSWLEEAIANFGWFGMLIGPLFIALLCRIGDSLQSRLLKSLSLLVAGLFLVVHLAAFMPLFL
IWFASVFTSAFIRLRARKRVSTVKAQPVDRYITV
>lcl|NZ_CP026520.1_prot_WP_156972850.1_4462 [locus_tag=PC41400_RS29955] [protein=hypothetical protein] [protein_id=WP_156972850.1] [location=complement(4960123..4960293)] [gbkey=CDS]
MKTKINIVLLVVSLAVVLVSPYLRDIFLSFGILAVLGMGAEYLFVRYSSGEESGLD
>lcl|NZ_CP026520.1_prot_WP_129112527.1_4463 [locus_tag=PC41400_RS22500] [protein=YolD-like family protein] [protein_id=WP_129112527.1] [location=4960584..4960760] [gbkey=CDS]
MILPEHKEAIQRQLRDLNKRTKPVLDEQEWELITRAMAYSYTEQQQITAVLFEPVFGN
>lcl|NZ_CP026520.1_prot_WP_042232531.1_4464 [locus_tag=PC41400_RS22510] [protein=lysophospholipase] [protein_id=WP_042232531.1] [location=complement(4961474..4962415)] [gbkey=CDS]
MAIPFEIVMDQSNVIRGDFYPASSIARGILIVSHGFKGFKNWGMFPYAAAKLARDVDVITFDFTHNGVGEDGQSFSELEK
FAKNTYSCELDDLSTLISVLTDEWDFKNKLLESDRLKPVVFYPHHSRGINLLNEEHQHRGSANPVNKLPIFLLGHSRGAG
VSLIYALDHPTEIAGVVSWNGITQVDIFDEACKEQMRSEGRAYIANARTKQQMPLDKVILDDLEARKDDFNILERMRGAT
FPVVLIQGEEDHANLRAGSEKLVENQPKVQWIQIPGANHTFNTIHPFAGTSAELEHAIEHTLQFLKSRVVPTE
>lcl|NZ_CP026520.1_prot_WP_009671780.1_4465 [locus_tag=PC41400_RS29960] [protein=hypothetical protein] [protein_id=WP_009671780.1] [location=complement(4962937..4963104)] [gbkey=CDS]
MQWVYWVKLYDSKFQAGVLAKRMEEDWWIYGYECPSEVEVFKSKKGRFGVRYSVY
>lcl|NZ_CP026520.1_prot_WP_042232529.1_4466 [locus_tag=PC41400_RS22530] [protein=phosphatidylglycerophosphatase A] [protein_id=WP_042232529.1] [location=complement(4963226..4963753)] [gbkey=CDS]
MKRQVHSREIKKVVMQRLSDRGASLEDIAEIVYAMQAPYHPGLSMDTCMKSIQAVLGKRELQHAILVGTELDMLAEQKKL
TEPLQTILDTDESLFGCDETLAIGSVFGYGSIAITTFGYLDKEKTGLIKRLDTKTGGAVHTFLDDLVAAIASAASGRIAH
KSRDEIERDEDNEYV
>lcl|NZ_CP026520.1_prot_WP_042232527.1_4467 [locus_tag=PC41400_RS22535] [protein=2'-5' RNA ligase family protein] [protein_id=WP_042232527.1] [location=4963918..4964430] [gbkey=CDS]
MMYGIAILPSAEINEFANSYRKRFDPYYRVISPHITVREKEEWTEEQLQAAISHLEDLSRTVSPFQITFNRFSSYFPVNN
VIYLALEQVEPVIDLHDKLCTGPLVDTEKAYNYHPHLTIGQKLGADELHDVLASVKNTPVDFSFHVDSFQLLGKQENGEW
SVIRSFPLQG
>lcl|NZ_CP026520.1_prot_WP_042232525.1_4468 [locus_tag=PC41400_RS22540] [protein=hypothetical protein] [protein_id=WP_042232525.1] [location=complement(4964553..4966352)] [gbkey=CDS]
MKSQWKTTVSSVFLSAALLASAVPALAAEPGKPAPTQASASAPTAYKLTNDLQVEMKSVLNEKTTDGTRLGAVVRMKNTS
SKVVRVPEYEVRLKTKDGVEFRLQASLGSSRSIQPKSQVELSYLATVDSGEGVELSELRWVDIDVYVYPKKETLQLSVPV
SDLVWNGTETSEIPSASVKKWGEAFTIPSLQTPLEFKTTNISRDTIGTKTNSVVQVLVTNPSTKRQTLPNIALEGKSGLQ
TYDSNRAESSLALEPGEKKYIHFVIPTELNTELEALNVVTAERYIDAQEKPVTYNVGRVHIMLPAKQADALPVHGAYAFG
DAMKMDPLNELIHPSMDVSLVELHMQQNEDDAFNTAIAKFKLTNKSGGPLNVPAFATDLVSSDGYTYNGNRQGTTTLRVL
PNASYVVTYSYILPASETGEGLKLALYETHGASAAAGAQASGAKSQEAQASAADTGASYKSLLTAYGVKLQPQPESKDEL
KLYPLTVKVNNWEISAMFNQQQNYNYRLKLLLDIEQEKDVLLDRTFSQLQFDMYDNGGTLINSVKSPFVGENKLQKGSNV
INFNADSMQLDFPVTVKVYESFVNDKGETAKRLLTTFKQ
>lcl|NZ_CP026520.1_prot_WP_042232523.1_4469 [locus_tag=PC41400_RS22545] [protein=efflux RND transporter periplasmic adaptor subunit] [protein_id=WP_042232523.1] [location=complement(4966379..4967647)] [gbkey=CDS]
MKRPLATIVKQSTKLVGIVALTTALAVGCSNKPLAGPQDTTQGQESQIKTVKTVKVTKEKISDPLEQVANVISSVQLEVI
SKTGGDVQQVYKKRGDMVKKGELLFRIDPSDLQLQKEQGQLGIKSAQAQLKQAKEELANAKTDLENNIVKAEQGIRDAEK
NFNKIRNDYDEGTVTKIQLEQEETKLNSLKLDLSTLKQKRKTLDNTDSLAAAQVAIETSGLSVKKAERELENTEVKAPVS
GVLTDFPVESGMTVGQGFKAGVIQQLNPILIKADLTEEAVNLVRDKKELNFYVPGTTELTKAPIKYISDVLDSQTKAYTL
ELEVSNADQKLKPGQKVQIQLTDEAEQTVAAIPTLSIVREGSDSFVFVLNGDTVEKRKIQLGRLKETMQEVISGVSENDQ
IVISGQHQLKDKEKVTTAAAAN
>lcl|NZ_CP026520.1_prot_WP_042232521.1_4470 [locus_tag=PC41400_RS22550] [protein=efflux RND transporter permease subunit] [protein_id=WP_042232521.1] [location=complement(4967679..4970786)] [gbkey=CDS]
MNKMTSFSMKNVAALFIIMLLVFAGGIFATTKLKVESMPDISFPMVVISTQYTAPPKDVLDQVTKPLEKAVAGFDGIKNL
SSTSSDNYSQITIELEQDIKPKNAKQDLEALVSNVKLPQQAEKPKVLTEGFSSQPVYYLAVYGNDGMNQAELDKIYKDTI
LPGFNTIKGIDHVDSVGNQEATLSIKLNENTINSYGLTPTDVSNSIRAALSASPAGTVEFNGTTQMVRVTGQIDTIYSLD
NLTLTTPNGSVLQLKDLGKIEAISESQFITRLDGKPAIGVQLYKTKSANAVNFADEADALMAKWEQTLPGVKFKSVYNSA
VEIKKSIHGMVQEGGMGALLASVMILLFLRNIRMTLIVLVSIPLSILITLLMMVPLNISLNIMTLGGMAIAVGRVVDDSI
VVIENIYSQLQKAQERGESVIKLATAQVAAAITSSTITTVGVFGPIAFVSGVIGEVFRPFALTLVVALLSSLLVALTVIP
MLAKLLVLKSKKLGNHDENHVGKFALRYKRVLEWSLNNRIKTLLFSGVLFIASLVLIVPQLAVSFMPEDQSARQMIFQIK
MPNDTSIETMDEKSKQLEQMMREAKDASGNPQFTYNESMVGYAFSKDVTPYRTMMFTEANENSNAEQVFNAYKEKILYEL
PKGSEVDGSLIAAGAGGSGTDFSYSLKGDDMMYLKQAAAQVKEKMKEFPDLHEVKDSLSDSKTEVEVTVDQNKARLYGLS
SAQVLEMVNSWIGVSKLGDLKFDNVTYETKVELDPQYKNSVNKIGTFLLKTPTGQTVQLNEIAKITQIDAPTGITRENQI
QTLKVTAKIKSADKGGVSGKVAAALNTLELPPGVSREVKGVSDDISSGFSEMFAAMGASIFIVYLVMVLAFGNASAPFAI
LFSLPLAVIGGLFGLLVTGESLNITSLIGFLMLIGIVVTNAIVLIDRVQQLREQGVEVRESLISAGLTRLRPIIMTAGAT
VFALMPLALGMSKGTIISKGLAVVVIGGLVTSTVLTLVVVPIIYEMIESFKRRSSNAFHRKPKKNKVDTVPAAKS
>lcl|NZ_CP026520.1_prot_WP_042232519.1_4471 [locus_tag=PC41400_RS22555] [protein=hypothetical protein] [protein_id=WP_042232519.1] [location=complement(4971062..4973194)] [gbkey=CDS]
MRNKMAARATALALILTLSTGGVLAATPAAAASGQPALGSVAAGASGSFELQNIQLLPDQNGNIVTFTVGVHNKSAADLA
LVDYWVRLKSKTGHTYTVRLMPQDKAVNKIGAHSSGQVSYYAKVGSGEKLQDLQVQFIKWDFSQPNSERVLGQISVPASY
TGVTPTGQNADFQVGSSELRASTEKFTNGKNEKYYLPAISLQLENIGGAAVTLPEVQFSVRTPEGLLYPLDAKNAKDLKI
SPKETKTVQLSGAIPLKAGEKGWELVITRIHPELKFNTLLAAYQLPAVSKPAGGESGKETQFMNKSGLYTAKLGGIYRLP
WEDQDILTANVTLSNKGEQSLPIPDMTAHFLLDNAVKVEAKLVRTGKVIGIDPGSSLSVEALGKIPYTYAFKEVKLVIQE
KESDTQVNDLLELTAKPDLQQLPFVGLGAKYAREEIGRKAQYTVAEINSYEGLSGSIVSAELEIENAEKRFAPVTGLVAH
FQLKDGTVFPASVSDIKKTVAPGGKARLNVWAALPKGTDMENVRLLIGDSVTLETGEGKPGDGQQTVKTDAYVRPVTFAL
PKENKDVKDKLKDLTLFPYTVSMSRINTSINNDVLTLKFNYDISKQLLTETAPEGHKLVLVFEDYKGNKSFEKAYELKNW
EPRDGKSAEQVTDNELRIGKKENFTIQLQDADLIFKQRFLDKYHLSVYHEFQGQRKLLAKQSIDWFAYSD
>lcl|NZ_CP026520.1_prot_WP_042232517.1_4472 [locus_tag=PC41400_RS22560] [protein=dipeptidase] [protein_id=WP_042232517.1] [location=4973459..4974829] [gbkey=CDS]
MKAKTESYASQHRERHLEELKELLRIPSISTLSEHKKDVHRCAEWLSGALTHAGLENVEIMPTAGLPIVYGDWLHAPGKP
TILVYGHYDVQPADPLDEWHSEPFEPVIRDGKIYARGATDDKAQLWIHVKAAETFLKAVGSLPLNIKFILEGEEEITSPS
LAPFVEKHAEKLKADAIVISDSPMLERGKPALLYGLRGVAGVEIEMLAARNDLHSGLYGGGVPNAVHALAGLLDSFHDED
GRVAVEGFYDRVIQLTEQERGELARVSQDEGKIREDLGLSELFGEKGYTFTERTTARPTLEVTGISGGFAGEGIKPIVPA
RASAKIACRLVADQRPEEIFELIERHVAVHTPSGLKASVKRVLQGNPFLTPADHPVMVAAAAALEEAYGEKTALTRGGGS
IPIVDVFERLLQAPVVMMGFGLPDENLHAPNEHFDLDNFDKGLQAVCVFWEKLDNQ
>lcl|NZ_CP026520.1_prot_WP_042232515.1_4473 [locus_tag=PC41400_RS22565] [protein=GNAT family N-acetyltransferase] [protein_id=WP_042232515.1] [location=4974898..4975407] [gbkey=CDS]
MQQQDTYKLRDAVPADLPAIVHIYNSTIPGRMVTADLEPVTVQSRENWFAEHNPETRPIWVLEKDGKIWGWLSFQSFYGR
PAYNGTAEVSIYLDEECRGRGLGRYLLEKAIAACPELGIKTVLGFIFGHNAHSLALFDRYGFAKWAELPRIAELDGIERD
LIIMGKRVG
>lcl|NZ_CP026520.1_prot_WP_042232513.1_4474 [gene=pyrE] [locus_tag=PC41400_RS22570] [protein=orotate phosphoribosyltransferase] [protein_id=WP_042232513.1] [location=complement(4975803..4976444)] [gbkey=CDS]
MNTAQSTEREIAASLLKIGAVALRPHEPFTWTSGIKSPIYCDNRLTMSHPEIRDAIAEAFADKIRSEYPQTEVVAGTATA
GIPHAAWVAQKLGLPMIYVRDKAKGHGKQNQIEGALAAGQKVIVIEDLISTGGSSLKAALAVNEAGGQALGVLAIFSYQL
DKGIQAFAEAGIPLDTLTNYSRLLEVAVEQGAIAQNDLELLRSWREDPASYGV
>lcl|NZ_CP026520.1_prot_WP_042232511.1_4475 [gene=pyrF] [locus_tag=PC41400_RS22575] [protein=orotidine-5'-phosphate decarboxylase] [protein_id=WP_042232511.1] [location=complement(4976500..4977234)] [gbkey=CDS]
METARAQSAGRIMVALDYASADEADTLIQELRGIPCYMKVGMQLFYAAGPRFVARLKDEGYNVFLDVKMHDIPNTVKGGS
ESVTRLGVDMFNVHAAGGIKMMEGALEGVDKALGGGAAKRPRVIAVTQLTSTSQMMLNEEIGIAGRVEDAVVRYARLARQ
AGLQGVVASPLEVSAVKEACGADFITVTPGIRPAGAALGDQNRVMTPAEAFTAGTDYVVIGRPITAAPDPRQALEQILDS
LEIR
>lcl|NZ_CP026520.1_prot_WP_042232509.1_4476 [gene=carB] [locus_tag=PC41400_RS22580] [protein=carbamoyl-phosphate synthase large subunit] [protein_id=WP_042232509.1] [location=complement(4977356..4980565)] [gbkey=CDS]
MPKNEKLKKILVIGSGPIVIGQAAEFDYAGTQACQALKEEGMEVVLINSNPATIMTDTNMADKVYIEPITLDFVTQIIRQ
ERPDGLLPTLGGQTGLNMAVELARAGVLEQENVKLLGTQLTAIEKAEDRDLFRDLMRELEQPVPDSTIVTTVPEAVDFAN
EIGYPIIVRPAYTLGGTGGGICHSEEELVETVASGIRYSPISQCLVEKSIAGMKEVEYEVMRDANDNCIVVCNMENFDPV
GVHTGDSIVVAPSQTLSDREYQMLRSASLKIIRALNIEGGCNVQFALDPFSFQYYVIEVNPRVSRSSALASKATGYPIAK
MAAKIAIGYTLDELVNPVTGQTYACFEPTLDYIVSKIPRWPFDKFINANRKLGTQMKATGEVMAIGRTFEESIHKAIRSL
EIGVHRLFLKETDLLDQETLELRLQKPDDERIFLLAEAFRRGYTIEQLQDLTQIDWWFLHKLDGMIRYEQEIALEELDNE
VLYDAKRMGFTDRAIAEIRSLAGRSTFTKEAEIREYRLNQNIKPVYKMVDTCAAEFEATTPYYYSTYEMEDEVLETEKQK
VVVLGSGPIRIGQGIEFDYSTVHAVWAIQAAGYEAVIINNNPETVSTDFNTSDRLYFEPLFFEDVMNVIEREKPIGVIVQ
FGGQTAINLAAPLSKAGVRILGSDLESIDKAEDRKKFEALLRKLDIAQPPGGTVTSVDQAVGTAAQFGYPVLVRPSYVLG
GRAMEIVYSDEELLSYMEYAVKINPEHPVLIDRYMLGKEVEVDAICDKETVLIPGIMEHVERAGVHSGDSIAVYPPQSLS
DEIKNQIIDITTKIGLELQVVGLVNIQFVIHNEQVYVIEVNPRSSRTVPFLSKVTNIPMANLATRVIMGQKLTEMGYQSG
LWPEDNFVSVKVPVFSFAKLRRVDTTLGPEMKSTGEVMGRDVNFAKALYKGLIGAGMKIPAAGAIIATVADKDKEEAVEI
FGGFQRLGYKIIATGGTAQVLEAAGVPVTTVNKLSEGSPNILDLIRSGEAHFVVNTLTKGKTPQRDGFRIRREAVENGVV
CMTSLDTVRALLHMLEAINFSSRSMPVYA
>lcl|NZ_CP026520.1_prot_WP_042232507.1_4477 [gene=carA] [locus_tag=PC41400_RS22585] [protein=glutamine-hydrolyzing carbamoyl-phosphate synthase small subunit] [protein_id=WP_042232507.1] [location=complement(4980552..4981730)] [gbkey=CDS]
MKAKARLLLEDGTLFTGKSFGAEGDSVGEVVFNTGITGYQEVLSDPSYCGQIVTMTYPLIGNYGISRDDFESVRPFVHGF
VVRRHEEAPSNWRAQYTLGNLLKEYGIIGISDIDTRMLTRKIRHHGVMKGLLTTSDRSVEELKEMLQTTPLMTTQVSTVS
TPHVFSAPGNRERVVLVDFGAKSGIVRDLTKRGCDVVVVPQDSTAEQIMRLAPDGVLLSNGPGDPKDVPHAVETVRQLLG
QVPVFGICLGHQLFALACGADTEKLKFGHRGGNHPVKDLISGRCYITSQNHGYTVKEESIAGTDLIVTHINNNDRTIEGL
KHNQYPAFSVQYHPEAAPGPFDSSYLFDDFLTMIRSHQTDNPQAPRQAQMLAAWKAEQSDNTSKGELHYAQK
>lcl|NZ_CP026520.1_prot_WP_042232505.1_4478 [locus_tag=PC41400_RS22590] [protein=dihydroorotase] [protein_id=WP_042232505.1] [location=complement(4981766..4983058)] [gbkey=CDS]
MGIWILNGKLLDASGKETTGHVLVENGKIAAVTAGQEQPDTTGHEVIEAAGKLVSPGFIDMHVHLREPGFEHKETIASGT
RSAARGGYTTIACMPNTKPVIDTVETVKLIQGKAENEGNGVKVLPYGCISRNQLGRELTDFEALKDAGVVAFTDDGVGVQ
SAQMMKDAMKRAAELGMPIVAHCEDETLVVGAPVSEGTFADKNGLKGIPNESEAIHVGRDILLAEATGVHYHVCHVSTEQ
SVRLIRLGKQIGVKVTAEVCPHHLLLSEEDIPGMDANWKMNPPLRSKRDVEAVIEALEDGTIDIIVTDHAPHSEEEKAKG
MQLAPFGIVGFETAFPLLYTKFVATGKWSLGFLIERMTSKPAEVFGLESGRLQVGSDADLTLVDLETEKEVDPAAFLSKS
RNTPFTGWKLKGWPVLTMVGGKIAWTETNE
>lcl|NZ_CP026520.1_prot_WP_084706656.1_4479 [locus_tag=PC41400_RS22595] [protein=aspartate carbamoyltransferase catalytic subunit] [protein_id=WP_084706656.1] [location=complement(4983187..4984098)] [gbkey=CDS]
MMGAVKTRTANHFIGLKGVSAPEIKGILDRADFWENHPDKVTNRLKGTFVSNLFFENSTRTRFSFEMAEKRLGAEVLNFS
AAESSVQKGESIYDTVRTLESMGIDAGVIRLKPVGLLAELAEKISVPLINAGDGNNEHPTQALLDLYTMRSHFGELKGLT
VSIIGDILHSRVARSNLWALREMGATVKFCAPANMQAPELSAYAPYVSMEEALQADVVMMLRVQLERHEKGMISSAEEYR
AKYGLTMNRLEQMPAHAIIMHPAPVNRNVELDDELVEHARSKIFTQIHNGVPIRMAVIERTLS
>lcl|NZ_CP026520.1_prot_WP_042232826.1_4480 [gene=pyrR] [locus_tag=PC41400_RS22600] [protein=bifunctional pyr operon transcriptional regulator/uracil phosphoribosyltransferase PyrR] [protein_id=WP_042232826.1] [location=complement(4984095..4984646)] [gbkey=CDS]
MSEHMHVLLDEMAVRRALTRIAHEILEKNKGVGNCMLIGIRTRGIYLAQRVAERIQEIEGVPVPVGEIDITSYRDDRVKV
RSVDTVPDGDNLNIQSKKVILFDDVLFTGRTVRAALDALIDLGRPEMIQLAVLVDRGHRELPIRPDYVGKNVPTSKLEKI
EVHLTEVDKHDRVVIIQQQRGNE
>lcl|NZ_CP026520.1_prot_WP_042232501.1_4481 [gene=dapD] [locus_tag=PC41400_RS22605] [protein=2,3,4,5-tetrahydropyridine-2,6-dicarboxylate N-acetyltransferase] [protein_id=WP_042232501.1] [location=4985070..4985780] [gbkey=CDS]
MTQMNTEEIIQFIKTSQKKTPVKVYVKGNLEAVDFGTGIQSFISGKSGVLFGEWTDIQAIIESNQQNIEDYVVESDRRNS
AIPLLDLKGINARIEPGSIIRDKVSIGNNAVIMMGAVINIGASIGEGTMIDMNAILGGRAQIGNMCHIGAGAVVAGVIEP
PSAQPCVVEDDVLIGANAVILEGVRIGQGSVVAAGAVVVEDVPEFTVVAGTPARVIKKVDDKTKSKTEILKELREL
>lcl|NZ_CP026520.1_prot_WP_042232499.1_4482 [locus_tag=PC41400_RS22610] [protein=N-acetyldiaminopimelate deacetylase] [protein_id=WP_042232499.1] [location=4985957..4987222] [gbkey=CDS]
MTSPFIELRRQLHRIPEPGFQEFKTQRMLLDYLQTLPQERMEIRTWRTGILVYIKGTNPSKRFGYRADMDGLPIPEETGY
PFQSEHPGYMHACGHDVHMSIGAGLVTQFAQRPMKDDLVVVFQPAEEGPGGALPMLASAELSDWMPEQMVGLHIAPEYPV
GTIAVRPGILFANTSELFIDLKGKGGHAAYPHLSNDMVVAASQLVGQLHTIVSRNINPLDSAVVTIGKMSAGTKQNIIAE
NARLEGTIRTLSAESMQLIKARIEGIVRGIESSYGCTATIDWGSNYLQVYNNETLTLEFMDWLQKTDTAPQAGGTDGSLA
DKSATGTVNGSEPAEAGQVQADSLSSWSEPQIPVRLYECTEAMTGEDFGYFLEKIPGFMFWLGAETPYGLHHARIEPDER
AIGVAIRTVDSYLRWKSAQSG
>lcl|NZ_CP026520.1_prot_WP_042232497.1_4483 [locus_tag=PC41400_RS22615] [protein=aminotransferase A] [protein_id=WP_042232497.1] [location=complement(4987299..4988462)] [gbkey=CDS]
MQPIINPQVKEIQISGIRKIANKVAAMPDALALTMGQPDFPTPAHIVEAAQKALQAGRTVYTPNAGLPELRKAAADFVGS
KYGLSYDGGSEVIVTTGASEALDITLRTILTPGAEVILPGPIYPGYEPLIRLCGGVPVYVDTRPYEFKMTAAAMEPHLTE
RTRCVVLGYPSNPTGQIMTKAELAELAALLGSRNLYVISDEIYSELVYGAEHASIAAMPGMREKTIVINGLSKSHSMTGW
RIGFTFAPEALTQHMVKVHQYNVTCASSISQYAAVEALTAGRDDALPMRQAYENRLEHVYGRLQAMGLEAVKPQGAFYLF
PSIARFGLGSEQFAYRLLEEQRVAVVPGDAFSPLGEGYIRLSYAYSQEVLDGALDRLEAFIRKLDRS
>lcl|NZ_CP026520.1_prot_WP_241688915.1_4484 [locus_tag=PC41400_RS22620] [protein=RluA family pseudouridine synthase] [protein_id=WP_241688915.1] [location=complement(4988540..4989466)] [gbkey=CDS]
MDETESEPTRWQVEAEQNGERIDKLLSDALDGVSRTQIQQWIKDGHARVDGRNVKPNYKLSGGQVVDLAIPDPEPLDIAA
ENIPLDVVYEDGDVIVINKPRGLVVHPAPGHYSGTLVNALMHHCKDLSGINGVMRPGIVHRIDKDTSGLLMIAKNDQAHA
SLAEQLKEHSVTRKYVALVHGNLQHDHGTIDAPIGRDSYDRKLYTVTDKNSKHAVTHFAVLERFDGYTLVELKLETGRTH
QIRVHLKFIGHPLVGDPSYGRSKGMTMDGQALHAQILGFTHPRTGERLEFEAPIPEDMADLLHRLRNL
>lcl|NZ_CP026520.1_prot_WP_042232494.1_4485 [gene=lspA] [locus_tag=PC41400_RS22625] [protein=signal peptidase II] [protein_id=WP_042232494.1] [location=complement(4989502..4989999)] [gbkey=CDS]
MYYYLLALIVFLIDQGTKWLVVKNIPLHDSIPVIGEFFQLTSHRNRGAAFGILQDQRWFFILATLIIVVGVVLYLNRTRK
AGQKLMSLALALLLGGAVGNFLDRLLFGEVVDFLQLHFQFSFFGKAVDYIYPIFNIADSAIVIGVILIFIDSIISWKNEK
RGTAS
>lcl|NZ_CP026520.1_prot_WP_042232492.1_4486 [locus_tag=PC41400_RS22630] [protein=TraR/DksA C4-type zinc finger protein] [protein_id=WP_042232492.1] [location=4990141..4990878] [gbkey=CDS]
MHILTSDQTDQLRQQLLIEKKELKEQLELDEHFGMSQAMGESTGELSTYDNHPADIGTEMFEREKDIALLENAERHLDQV
NFALERLAAGTYGICIECGRTIPYERLSAIPTTQYCLEHVPNRHASDRRPVEEEFLAPPFGRTSLDERSDQNQFDGEDAW
QIVESWGNSDSPAMAEDPNIGDYNAMYVEADELDGYVEPFESFLATDIYGNHVSVIRNKAYRSYWDNQEGDPLLEPDYTS
EETQS
>lcl|NZ_CP026520.1_prot_WP_042232490.1_4487 [locus_tag=PC41400_RS22635] [protein=DUF5665 domain-containing protein] [protein_id=WP_042232490.1] [location=complement(4990916..4991347)] [gbkey=CDS]
MDNLNEERNKRNERNEPVKARPPEEHAEPEHPDLPANASLTGDEEKRLIHHIHEKVSEIATNIERTQIADYVTLMNKPRR
LIFANLIAGISRGVGLGLGITVFAGTILYALKALGALNLPIIGTYIADLVEYVQFQLEGRRFY
>lcl|NZ_CP026520.1_prot_WP_042232488.1_4488 [gene=ileS] [locus_tag=PC41400_RS22640] [protein=isoleucine--tRNA ligase] [protein_id=WP_042232488.1] [location=complement(4991417..4994179)] [gbkey=CDS]
MDYGKTLQLLKTDFPMRGNLPQAEPKMQAHWAENDIYAKVQESRKGREKFILHDGPPYANGDIHIGHALNKILKDIIVRF
KTMQGYDAPYVPGWDTHGLPIEQAIANSGKVDRKKMSVAEFRKYCEEYALQWVERQKEQFKRLGIRGDWDHPYITLQPEY
EAEQIRLFGEMVKKGYIYKGLKPVYWSPSSESALAEAEIEYKEKTSPSIYVAFDVADGKGKLPQDAAFVIWTTTPWTLPA
NLGISVHPDFTYAVVEMSGKKYVLAEGLLESAAKAIGWTDYSIVSTVKGSELENVTCKHPFYDRESLVMLGEHVTLEAGT
GCVHTAPGHGEDDFIVGQKYKIGVLCPVDDQGHFTAEAPGLEGVFYEKGNKTILEWLENSGHLLHASSINHQYAHDWRTK
KPVIYRATEQWFASIDKFRDEMLEEIKKIKWTPNWGEIRLHNMIADRGDWCISRQRVWGVPIPIFYCRSCNHPIVNDQTI
EHVAKLFEKEGSNAWFTLSEKELLPEGLSCPECGHGEFRKETDIMDVWFDSGSSHAGVLAKRPELQWPADLYLEGSDQYR
GWYNSSLITGVAVKGQSPYKGILSHGFTLDGEGRKMSKSLGNTVDPSQVCNKLGADILRLWVSSVDYQSDVRISDNILSQ
ITEVYRKIRNTLRFLLGNLSHFDPVKDRVDFAQMSELDQYAVIRLNRMTEKVIKAYDAYEFHVVYQAVHHFCAVEMSSFY
LDIIKDRLYADAADDSARKASQTVLYDALLAITKLIAPILPHTADEVWKYIPGVELDSVQISELPEIRTYVYNQELEQKW
DGFLNIRDEVLKALEEARQNKTIGNSLSAAVDLYPSEAAFELLSGFDKLDTLFIVSAATLHNPSEKAPEGSYEAQDLAVQ
VSVAEGEKCERCWIVTPEVGQNKEHPTLCARCAEVVSRLE
>lcl|NZ_CP026520.1_prot_WP_009671797.1_4489 [locus_tag=PC41400_RS22645] [protein=DivIVA domain-containing protein] [protein_id=WP_009671797.1] [location=complement(4994618..4995103)] [gbkey=CDS]
MPLTPLDIHNKEFARSFRGYDEDDVNEFLDQVIKDYEAVIRENKELQNQILALQERLDHFSNIEETLSKTIIVAQEAADE
VKNNAKKEAQLVLKEAEKNADRIINESLTRSRKVSMETEELKKQASIYRTRFRTLLEAQLELLQDEGWKTLDHDKVLELE
K
>lcl|NZ_CP026520.1_prot_WP_042232485.1_4490 [locus_tag=PC41400_RS22650] [protein=YlmH/Sll1252 family protein] [protein_id=WP_042232485.1] [location=complement(4995213..4995992)] [gbkey=CDS]
MAKEIYGHFHPDEHAFVDRAQEWVENAAHHKVKRTDFLDPRQAFILRTLVNRNPDVNCRLDGGYPDAERQRAILGPDYLN
LSGEDMGLRVLSVESGDSKFQALSHGDYMGAILALGMKRDKVGDINVSEQGCHFVVAEEAADYLHLNLSQVHRVHVTTEI
LPIDKLRPSAVKLEEMSLSVASLRLDGIVSDVFRLSRAKVLIPIGAGKCKVNWKPEENPSVPLQEGDVVSLQGFGRFRVL
EIEGVTKKGRTRMKVGKYV
>lcl|NZ_CP026520.1_prot_WP_174446833.1_4491 [locus_tag=PC41400_RS22655] [protein=YggT family protein] [protein_id=WP_174446833.1] [location=complement(4995992..4996261)] [gbkey=CDS]
MNIYGLIDTLAQIYFYMIFGYVLLSWFPNARESFIGELLAKLVEPYLSAFRRFIPPIGPLDISPIVAMGVFWLVVAGLKV
VVGYIVGIF
>lcl|NZ_CP026520.1_prot_WP_009671786.1_4492 [gene=sepF] [locus_tag=PC41400_RS22660] [protein=cell division protein SepF] [protein_id=WP_009671786.1] [location=complement(4996298..4996723)] [gbkey=CDS]
MGVMNKFMNFIGLQEEEEVVERERIVETPEEPETPVQEQRTKQSKNNIVSIHSQKNSRVVLHEPHSFDDSQEIADHLRSR
RTVVVNLQRVRPEQAVRIFDFLNGTVYALNGSISKIGPNIFLCAPDSVEIQGSISDFLHEA
>lcl|NZ_CP026520.1_prot_WP_042232480.1_4493 [locus_tag=PC41400_RS22665] [protein=YggS family pyridoxal phosphate-dependent enzyme] [protein_id=WP_042232480.1] [location=complement(4996760..4997443)] [gbkey=CDS]
MNLQARIEDVHHRIEAACRRSGRSPEDINVIAVTKYVSLEATREALDHGLTQVGENRWQDAKEKWEALGDRGDWHFIGHL
QTNKVKDIIGKFAYIHSLDRASLAKELEKEAAKRDLFVKCFIQVNVSGEESKYGLPPEELESFAKDLKQYPHLQIVGLMT
MAPLEAEAEATRPVFRKLRELRDALNAERILPEPLRHLSMGMSGDFEVAIEEGATWVRLGTVLVGKS
>lcl|NZ_CP026520.1_prot_WP_042232478.1_4494 [gene=pgeF] [locus_tag=PC41400_RS22670] [protein=peptidoglycan editing factor PgeF] [protein_id=WP_042232478.1] [location=complement(4997492..4998346)] [gbkey=CDS]
MEPFVRKKSNSGEPELFVLERLQERFPFVQAGFTSRHGGVSSAPFDSLNLGLHVNDADEDVVENRKRLAAALGVPFEALT
FGEQVHGKEVAVIEKKQMGSGRISRADAIQGKDAFITAGKGIMLCALYADCVPLYLIDPVNRVVGIAHAGWKGTVQNIGA
AALASMTRSFGSRPEDVHAVIGPSIGGCCYEVDDFVAGRVYESHLDTGISKKQTETVVIPRKEGKYQLNLQECNRQFLEK
AGILSSHIEVTELCTSCSTDLFFSHRKEGGSTGRMAAWIGLRND
>lcl|NZ_CP026520.1_prot_WP_042232476.1_4495 [locus_tag=PC41400_RS22675] [protein=YlmC/YmxH family sporulation protein] [protein_id=WP_042232476.1] [location=complement(4998544..4998840)] [gbkey=CDS]
MKISDFQTKDVINIVDGKKLGQISDLELELRQGRIDSIVVPNSSKFFGLFGGGTDLIIPWKNIVKIGADVVLVKLEEAKT
YRAVEESDSADYNRSPRG
>lcl|NZ_CP026520.1_prot_WP_009671832.1_4496 [gene=sigG] [locus_tag=PC41400_RS22680] [protein=RNA polymerase sporulation sigma factor SigG] [protein_id=WP_009671832.1] [location=complement(4999029..4999811)] [gbkey=CDS]
MTRNKVEICGVDTAKLPVLTNVEMRELFTNLQTKNDRSAREKLVNGNLRLVLSVIQRFNNRGEFVDDLFQVGCIGLMKAI
DNFDLGQNVKFSTYAVPMIIGEIRRYLRDNNPIRVSRSLRDIAYKALQVRDSLTNKNSREPTIYEISEVLNVPKEDVVFA
LDAIQDPVSLFEPIYHDGGDPIYVMDQISDERNKDMSWIEGIALREAMRKLGSREKMILSMRFFEGKTQMEVADEIGISQ
AQVSRLEKSAITQMQKHVKS
>lcl|NZ_CP026520.1_prot_WP_042232474.1_4497 [gene=sigE] [locus_tag=PC41400_RS22685] [protein=RNA polymerase sporulation sigma factor SigE] [protein_id=WP_042232474.1] [location=complement(5000083..5000805)] [gbkey=CDS]
MRIKWKLQMLIFYYKILIFFGLKGEEIYYIGGSEALPPPLTREEEEYLLGKLPSGDAAIRAMLIERNLRLVVYIARKFEN
TGINIEDLVSIGAIGLIKAVNTFDPEKKIKLATYASRCIENEILMYLRRNSKIRTEVSFDEPLNIDWDGNELLLSDVLGT
ENDTIYRNIEEQVDRKLLHKALDKLTDRERTIMELRFGLQDGEEKTQKDVADQLGISQSYISRLEKRIIKRLRKEFNKMV
>lcl|NZ_CP026520.1_prot_WP_042232472.1_4498 [gene=spoIIGA] [locus_tag=PC41400_RS22690] [protein=sigma-E processing peptidase SpoIIGA] [protein_id=WP_042232472.1] [location=complement(5000835..5001758)] [gbkey=CDS]
MVVYADLIFLVNVVMDAAVLLVTARTRRLSFKPWKLVVSSALGGSYVLLMFVPSLSFLFTFAAKIIMSLCMVWIAFGYGS
LQHFLRNTGTFYVVNFVAAGGVFGLNYVFQTSGDVMNGLLFSQTGAARFSIRIETWAVIVLGAGSLVLHRLVMAGVKKKE
ELAVHLAEVSVIVDGADFRCTGLIDTGNQLYDPLTRTPVMVMEASQLASMLPESWLKRIRAAEVEQIITAIGTEEDFVWQ
DRLRLVPYRGVNRNTQFMLAIKPDKVVITYGSGQVETTKVLVGLDGGQLSGDGAYHAIIHPTLVAAG
>lcl|NZ_CP026520.1_prot_WP_042232471.1_4499 [gene=ftsZ] [locus_tag=PC41400_RS22695] [protein=cell division protein FtsZ] [protein_id=WP_042232471.1] [location=complement(5001963..5003111)] [gbkey=CDS]
MFEFDLDMEQLAQIKVIGVGGGGSNAVNRMIENGVQGVEFITVNTDAQALHLAKSEHKLQIGDKLTRGLGAGANPDVGKK
AAEESRDLIMNTLKGADMVFVTAGMGGGTGTGAAPVIAEIAKECGALTVGVVTRPFTFEGRKRAAQAEQGIAALKEKVDT
LIVIPNDRLLEIVDKKTPMLEAFRQADNVLRQGVQGISDLIAVPGLINLDFADVKTIMTERGSALMGIGIATGENRAAEA
AKKAIMSPLLETSIEGARGVLMNITGGTNLSLYEVNEAADIVASASDIEVNMIFGAVINENLKEEIMVTVIATGFQDNPQ
QQQPPQPNNRRQGGVQSQQQQQQETPAETQQRVNNLRPFGGQPGDLEVPTFLRNRGRNNLDK
>lcl|NZ_CP026520.1_prot_WP_009671845.1_4500 [gene=ftsA] [locus_tag=PC41400_RS22700] [protein=cell division protein FtsA] [protein_id=WP_009671845.1] [location=complement(5003334..5004587)] [gbkey=CDS]
MSNNDIIVSLDIGTSKVRAIIGEVVNGTINIIGVGSADSEGIRKGAIVDIDQTVQSIRSAIDHAERMVGIQISDVYVGIT
GNHIALQDSHGVVAVSNEDREIGEEDIERVIQAARVIALPPEREIIGVVPNQYLVDGQEGIHEPRGMIGVRLEVESTIIT
GAKTGIHNLLRVVEKSGLKVAGLILMSLASGQLALSKDEKSIGTVLVDIGAGSTTIAVFDQGNLVATSTLPIGGDFITND
ISIGLRTQMDIAEKIKLKYGCATIQDSAPDQVFKVNRIGSQVDKEFSQVDLANIIEPRVQEIFHLIRAEASRLGYSDPAG
GYVLTGGTVSLPGTLVIAQEELAAPVRIAVPDYIGVRDPAYTSGVGIIQFVCKFLRNRPVTAPKKQTANKPASVKTPQTE
ASPGWFDKVKNFLKEFI
>lcl|NZ_CP026520.1_prot_WP_241688719.1_4501 [locus_tag=PC41400_RS22705] [protein=FtsQ-type POTRA domain-containing protein] [protein_id=WP_241688719.1] [location=complement(5005048..5005908)] [gbkey=CDS]
MPEENKVPVLPNPKPRRTRANRKLLTLLFLFFVTILAVLFFRSSLSRVSEIKIAGNELLSSEEIGQAAAVRKGDHFFAVG
SESVRGRVAALRMVQSVEVSKHFPGQISIVVHEYPRVAFQFGADGVKQAVLADGSTVPVVKEDMPVDKPILTGWKENDPN
RIALCKALAAIPAGYLSDISEIRPDPSEAYKDKIKMYTRTKFEVVTTVGYLPNKIMNLSAYITSMQGSEVNSGILMLLET
DDHLPLDTSAAGNKGKAGTGSTKKPDPTPKPGGSVKPTSTPQKDTR
>lcl|NZ_CP026520.1_prot_WP_042232466.1_4502 [gene=murA] [locus_tag=PC41400_RS22710] [protein=UDP-N-acetylglucosamine 1-carboxyvinyltransferase] [protein_id=WP_042232466.1] [location=complement(5006003..5007289)] [gbkey=CDS]
MEKLVIEGGIPLSGAVRIQGAKNAALPILAASIMASGTHTLRNVPKLLDIEVMLGILRSLGCGAEHKEDTVVLDTTGLNS
THVPEELMKQMRSSIFLMGPLLSRFGTVQVYQPGGCAIGERKIDLHLRGLQALGVTIEETGSTIICTAEDLKGADITLDF
PSVGATENIMMAAVLADGITTISNAAREPEIQDLQNFLNAMGADIIGAGTDTITIRGVRSLHACTYRIIPDRIVAGTFMV
AAAATKGSVALEGVCPQHLNAAIHVLRRAGVQITVHDDIINVHAPIRPRAVERIVTAPHPSFPTDLQSQVMVLLSLADGI
SIMKETIFESRFKHVDELTRMGADIRVDFNSAFIRGVPRLYGATVEATDLRAGAALVIAGLAAHGTTVVEQIHHVDRGYD
RIEEMLTRLGARIRRQAPVPDQAPVTGE
>lcl|NZ_CP026520.1_prot_WP_042232464.1_4503 [gene=murB] [locus_tag=PC41400_RS22715] [protein=UDP-N-acetylmuramate dehydrogenase] [protein_id=WP_042232464.1] [location=complement(5007305..5008210)] [gbkey=CDS]
MHQLITDLQAAQVGEVLKNEKLAPYTTWKIGGPADLLVIPQDKQKLVETIRLLGQHKVPWTVVGRGSNLLVGDKGIRGVV
IKLSRALETLRFDGSTVYAGGAYSFIKLSVMTGKEGLTGLEFAGGIPGSVGGAVYMNAGAHGSDVSRILKQAEVLLDTGE
LIIMQNSDLQYAYRHSVLHTLPGIVTEAVFELQPGDRKEIAGAMAAYKDRRLRTQPLQLACAGSVFRNPPGNHAAKLIEE
AGLKGTRVGGAEISPLHANFIVNTGDASAEDVLALIALAQKTVREKSGVELQPEVLVVGER
>lcl|NZ_CP026520.1_prot_WP_042232462.1_4504 [gene=murG] [locus_tag=PC41400_RS22720] [protein=undecaprenyldiphospho-muramoylpentapeptide beta-N-acetylglucosaminyltransferase] [protein_id=WP_042232462.1] [location=complement(5008420..5009550)] [gbkey=CDS]
MRIVFSGGGTGGHVYPALAIGEQCLQDHPDSKFLYIGTSSGLERDIVARSGLDMPFEAVEITGFRRKLASLDNVKTVLRF
LRGVGRAKSFLREFKPDAVVGTGGYVCGPVLYAAAKLGIPTLVHEQNAIAGLTNRFLSRYVSTVAVSFRGTEGIFPQAKK
VVYTGNPRATSMVNADAARGFSSLGLTPDARLVVVVGGSRGAKAINEAMVELAPLVHELPGVQFVFGTGQPYFESTAQRI
QEAAGSVPVNLQVLPYIHNMPEVLAASKLVVNRAGASTMAEITALGIPSILIPSPNVTNNHQEHNARSLSDEGGSVTLLE
KDLSGRVLFDQIGKIMEDSVRWEHMAKQSAKMGQRDSAALVVRELEQIVSAGQRRR
>lcl|NZ_CP026520.1_prot_WP_042232460.1_4505 [gene=spoVE] [locus_tag=PC41400_RS22725] [protein=stage V sporulation protein E] [protein_id=WP_042232460.1] [location=complement(5009556..5010653)] [gbkey=CDS]
MAKARSAPDVWILVSTLLILTIGVIMVYSASGVLAFREFGDWFYYLKRQLLFAVLGIVAMFFTMNTDYWVWKKYAKLGLL
ICFGLLLAVLIPGIGVVRGGARSWLGIGSFGIQPSEFMKIGMIVFLAKLLSDRQSQITLFTKGLLPPLGLVGLAFGLIML
QPDLGTGAVLVGASLLVIYTAGARILHLSYLAMAGVAGFVGLIIAAPYRLQRITAFMDPWSDPLGAGYQSIQSLYAIGPG
GLAGLGLGMSRQKYSYLPEPQTDFIFSIIAEELGFIGGAAVLILFTILIWRGMRTAITAPDTFGSLVAVGIVGMIAVQVI
INIGVVIGMFPVTGITLPLISAGGSSLTLMLTSIGVLLNISRYSR
>lcl|NZ_CP026520.1_prot_WP_042232458.1_4506 [gene=murD] [locus_tag=PC41400_RS22730] [protein=UDP-N-acetylmuramoyl-L-alanine--D-glutamate ligase] [protein_id=WP_042232458.1] [location=complement(5010763..5012151)] [gbkey=CDS]
MKHPSEYRGEKVVVLGLARSGVAAAKFFHEMGADVTVNDRKTRETSPEADELEALGISVVCGGHPPELVGEDTALIVKNP
GIPYGAPPVVQAAERGIEVVTEVEVAYHICAAPMIGITGSNGKTTTTTLIGLMLEAGKLNPIVAGNIGRALTEAAAEAKP
SDWMVVELSSFQLKGTKEFRPKIACLLNLYETHLDYHGSMEDYVASKAKLFANQKQDDVAVLNADDEVCRKLAGEVSSEV
LLFSRQRELDKGVYLSGDGVLTYRDAEGGTYPILPAKELGIRGSHNIENALAASAVALSAGVPVEAIADVLRTFRGVEHR
LEFVGEFGGVAYYNDSKATNPAATTKTVESFEQKVILIAGGLDRGSDYKEMLPLLASRVKAVVTLGETREKINRVAREAG
LAAVETVDTANSTEAIGEAVHAASKLAGEGDIVLLSPACASWDMFASYEERGRMFKESVHNL
>lcl|NZ_CP026520.1_prot_WP_042232456.1_4507 [gene=mraY] [locus_tag=PC41400_RS22735] [protein=phospho-N-acetylmuramoyl-pentapeptide-transferase] [protein_id=WP_042232456.1] [location=complement(5012155..5013117)] [gbkey=CDS]
MELKVILFTMGVAFLLALIMGPLFIPLLRRLKFGQTIRTEGPQSHQKKQGTPTMGGIIILIALAIAALRFSDSNTELWIL
LIASLGYGLVGFLDDYIKIVFKRSLGLTAKQKLIGQLVVSAVVCVLLVTSNHSTTLFVPVLDVSFDTGWLYFPLMTILML
GASNAVNFTDGLDGLLAGTSAIAFGAFTIIAMNNTQPEAAIFSAAMVGGVLGFLVFNAHPAKVFMGDTGSLGIGGGLVAV
AIMTKAELLLAVIGGVFLIEILSVVIQVISFKTTGKRVFRMSPIHHHFELVGWSEWRVVTTFWAAGLVLAALGLYMNEVL
>lcl|NZ_CP026520.1_prot_WP_042232454.1_4508 [locus_tag=PC41400_RS22740] [protein=UDP-N-acetylmuramoyl-tripeptide--D-alanyl-D-alanine ligase] [protein_id=WP_042232454.1] [location=complement(5013293..5014717)] [gbkey=CDS]
MKKLPLSTIAELAGAKLLIPPGADNGAGGIVIDAVSTDTRTIAPGSLFVPLVGERFDGHEFAPDAAAKGASAALWLESRQ
EGRPEGLPLLLVDDTLTALQKLAHAYRKTLNVKVVGITGSNGKTTTKDMMGAVLGTVYNVHRTKGNLNNHIGLPLTLLQL
DDTTEVAVVEMGMSGRGEIELLSEIAEPDIAVITMIGDAHLLQLGSREEIARAKTEILSGLKDGGTYVFHGDEPLLPLVL
PEMKQPARMSQIRFGRGHTNDLYPTLIQMDGEGTHFSINPRDTESDKDAQKYYIPLLGEHNVDNALAAIAVGTALGVPDA
DIRRGLAGMQMTGMRIEKRLTPGGVTVLNDAYNASPSSMKASIRLVEQLEGYPRKYVVLADMLELGEDEIRYHREIGSLL
SPDALAGVYAYGSLGRHIAEAAGESFGPERVRWFEDKAELTAALHAALKSGDLVLIKGSRGMKLEEVAEALLAK
>lcl|NZ_CP026520.1_prot_WP_042232453.1_4509 [locus_tag=PC41400_RS22745] [protein=UDP-N-acetylmuramoyl-L-alanyl-D-glutamate--2,6-diaminopimelate ligase] [protein_id=WP_042232453.1] [location=complement(5014714..5016201)] [gbkey=CDS]
MELKELASQLLICQLIGEETTEISGIATDSRQVRPGDLFVCVPGFVSDGHDFAPKAIELGAAALVTERKLELDVPQLIVK
DARYAMSVLSACVYGYPSRKMKVIGITGTNGKTTTSFLVEHILRDQGFRTGLMGTISAKIGSEFIPMERTTMEASDLQRT
FRLMADEGTDYCVMEVSSHALDLGRVKGVRFRGGVFTNLTQDHLDYHETMDKYESAKGLLFSRMGNEFTGDAGDKQYAVL
NADDPSSERLAKLTAAEVITYGVNRDCDVKAANISITAKGTEFDVVSFAGTEPMRMKLVGLFNVYNALAAISVSLAEGLS
LSQIRESLGSMPGVDGRMEIVDEGQEFLVLVDYAHTPDGLRNALGSIREFAQGRIKCVFGCGGDRDRTKRPLMGKVTGEL
SDDVYVTSDNPRTEDPGAILLDIVPGLVEAGVDESRYKLLVDRPEAIQKAIEEAGPQDVVLIAGKGHETYQDIAGVKHDF
DDRLIARAALRRLQS
>lcl|NZ_CP026520.1_prot_WP_042232451.1_4510 [locus_tag=PC41400_RS22750] [protein=stage V sporulation protein D] [protein_id=WP_042232451.1] [location=complement(5016302..5018254)] [gbkey=CDS]
MRVSSFTVRRRLFAALILGTVLFAALIIRLAYVQLWLGPGLSAKAEDSWRREIPFMANRGEIWDRNGEKLAYNISTPSIM
AIPVQLKDPNATAAKLAEVLQAPVDSIYKQITKRERINEIKPAGRKITLEKAQEIKALNLPGIVVAGDNRRYYPFGSLAA
HILGFTGTYNQGLTGVESKYDKLLTGIKGNVSYLADAANRKMPGSTEQYSEPKDGLNLQLTIDQHLQAVLERELDQAMVK
YQPKHAIAIMMDPNNGEILAMGSRPTYEPGNFGKYPVETYNRNLPIWMTYEPGSTFKIITLAAALEEKKVDLKNERFHDP
GAIEVGGARLRCWKRGGHGSESFLEVVENSCNPGFVVLGQRLGKETLFDYIKRFGFGRKTGIDLGGEENGIMFKPSQVGP
VELATTAFGQGVSVTPIQQITAVSAAINGGKLFRPHVAKSWIEPETGRVIETVKPELVDTVISEATSKQVREALESVVAK
GTGGNAFIDGYRVGGKTGTAQKVVNGRYSPNEHIVSFIGFAPADNPKVVVYAAVDDPQGIQFGGLIAAPLVKNIMQDALR
YMKVEPDKNQLTKKYKYGETPIVEVPDLVGATVQDIYEDLNSDFLIAKSGTGNTVISQAPKPGTRVDRGSTIRVYLSNPP
EGAMDKKEGR
>lcl|NZ_CP026520.1_prot_WP_042232449.1_4511 [locus_tag=PC41400_RS22755] [protein=penicillin-binding transpeptidase domain-containing protein] [protein_id=WP_042232449.1] [location=complement(5018340..5020673)] [gbkey=CDS]
MVKKMKMRSLLIGGFFTLLFIVFLGRILWIQGVSASKLLNEAEKRWETDRIVPAERGKILDREGRILAEEVPAYSVSLNP
KVIYQTGLADDITAGLTEILKDNSDSPNLSDKIKGLATKKRAVDPAAAQSNGANAAADDKKDNKKDAKKEPKPEDAYLVN
VEVNNEGKNIDKAKADEVQQLVKKLQQQLNDKDKKKYGTQADVGIYLTTGKKRYYPGNSLAAHILGYTNREGKASMGIEA
QLDEQLKGTPGSLSYNSDKNGVELVQSKSVYVPPVNGNYVKLTIDKNIQYYMENALSKVMDQWHPKSLMAIAADPKTMEI
LGMASYPNYNPNEFWKGGDQTNHNVQSQYEPGSTFKLVTLAAAVEEGVFNPGATFKSGSVQVPGRKPVYDHNKIGWGPIT
YLEGLKRSSNVAFVNLGYKMLGVERLKDYIDRFGFGKKTGIDLYGEVAGQTNLAYEVDYAAATYGQALTATAIQEAAAYG
AIANGGKLMWPHVVKEIIDPKTQQPIQKIEPKVVGQPVSAATAKKVGEYLEQVVADQVKGTGKDAYIEGYRIAGKTGTAN
KVVPGQKGYAANKWVISFAGFAPVEDPQFVLIIIADEPELGNNYLLGHYVAAPAFKEIMQNSLRYLRIPSSKQQAQVAPV
QTSTQKAVDFGNMTAAQAKAAAERAGVDIETVGQGAKVVAQYPKAGTEIGPYQRVYLAMQAPAEIAVPDMKGKALRDALE
LCALLGIRTASTGEGYVVSQVLQGEGSDRVLTLQLQPLGGEIVTPPEEKQVVPEKSE
>lcl|NZ_CP026520.1_prot_WP_042232447.1_4512 [gene=ftsL] [locus_tag=PC41400_RS22760] [protein=cell division protein FtsL] [protein_id=WP_042232447.1] [location=complement(5020709..5021131)] [gbkey=CDS]
MPGYIQGNLAEKSTATSQRVTVRETKKVIVKNKPLPVGEKLMYLLLVIVCVLVATGIVWRYTTIYQINSEIKTTKTEITM
LKAENDANQQEVQKLKDPERLEAIAKEKGLKPAGDNQVRQVTPGSGQASIDKKSSVAAGR
>lcl|NZ_CP026520.1_prot_WP_042232445.1_4513 [gene=rsmH] [locus_tag=PC41400_RS22765] [protein=16S rRNA (cytosine(1402)-N(4))-methyltransferase RsmH] [protein_id=WP_042232445.1] [location=complement(5021145..5022095)] [gbkey=CDS]
MFHHITVLKEEAVDGLHIQPDGVYVDCTLGGAGHSSLIASRLGPEGLLIALDQDDAALANAEHILAPYRDRVKIVKTNFR
RIEEVLRDEPRAFRDGVPQVNGILFDLGVSSPQLDEAERGFSYHQEAELDMRMDRSTELTAKIIVNEWPEAEIARILFVY
GEEKFSRRIAGAIVRARADGPIETTSQLVELIKEGIPAAARRTGGHPAKRSFQALRIAVNDELGAFEDALHQSVRCLAPG
GRISVITFHSLEDRICKQIFSSYVGRCTCPPDFPLCVCGNTGVVKLVNRKPITAGSAELEHNTRARSAKLRIAEKL
>lcl|NZ_CP026520.1_prot_WP_009671842.1_4514 [gene=mraZ] [locus_tag=PC41400_RS22770] [protein=division/cell wall cluster transcriptional repressor MraZ] [protein_id=WP_009671842.1] [location=complement(5022140..5022577)] [gbkey=CDS]
MFMGEYQHNIDEKGRLIIPAKFREALGVSFVVTRGLDQCLFVYPKSEWSVLEQKLKALPLMKADARAFTRFFFSGATESE
LDKQGRVNIPNNLVEHAKLEKDCVVLGVSNRVEIWSKSVWESYFEQSEQSFNEIAEKLVDFDFDL
>lcl|NZ_CP026520.1_prot_WP_042232444.1_4515 [locus_tag=PC41400_RS22775] [protein=DUF4349 domain-containing protein] [protein_id=WP_042232444.1] [location=complement(5022762..5023892)] [gbkey=CDS]
MKHNHAVRGRKRESDGDGKRDSFRIARRGLIVALLVGASLTGCSAADKGASTADQKSGTAATASLNTEMMKKEGASKQAD
GNSASPAPAAVNPSATPAAADGFDRKVIYKGNLVAEVEDYAKAQRALRELVTASGAYILQFTEDNTASEIGGTFTIKVAA
GGFSSLLDGIEKISPTRQRNITGQDVTEEYVDLTSRLKAKEVVEARLLAFMEKATKSDELLAFSAELGKVQEEIERIKGR
MRFLDQNVEYSTVELKLVQPKEGVLLKGAGTPFGKQIGSTFTKSIDFIFTLLKGLVIFVTALLPVAAIALVIGLPIYFGV
RSRNRRRMKQVEAQKFTQAFESRQPDSSFHEKGKALEDNTDPIRQDESTVPEKEKQ
>lcl|NZ_CP026520.1_prot_WP_042232442.1_4516 [locus_tag=PC41400_RS22780] [protein=adenosylhomocysteinase] [protein_id=WP_042232442.1] [location=complement(5024089..5025351)] [gbkey=CDS]
MNAVERSIVADQSLAPEGHLKIDWVKAHMPVLNRIREQFEKEQPFRGLKVAISLHLEAKTAYLAKVVQAGGAEVTITGSN
PLSTQDDVCAALAEDGITVFAKYNPDPKEYKELMIKTLETKPDLIIDDGGDLVTILHSERRDLLAGVRGGAEETTTGILR
LKALDKEGRLEFPMVAVNDAFCKYLFDNRYGTGQSVFDGINRTTNLVVAGKTVVVVGYGWCGKGVAMRAKGLGAQVVVTE
IDAIKAVEAYMDGFTVLPMSEAAKIGDYFVTVTGNRDVIRGEHFEVMKDGAILSNAGHFDVEVNKPELAAQSSSVRTVRR
NIEEYQLKDGRKIYLLAEGRLVNLAAGDGHPAEIMDMTFALQALSLKYVNDRYEEIGKTVVNVPYELDEQVARYKLESLG
ISIDRLSDEQKAYLQSWQEH
>lcl|NZ_CP026520.1_prot_WP_042232440.1_4517 [gene=bshC] [locus_tag=PC41400_RS22785] [protein=bacillithiol biosynthesis cysteine-adding enzyme BshC] [protein_id=WP_042232440.1] [location=complement(5025403..5027037)] [gbkey=CDS]
MPWKAYQWKKSQPLTEDYIHHFDRVSALFDYNPWDKGSLHRRAEAVRHTEGNRADRKRLAEVLLEYNRRIGNQETALSAA
AGLAEENSLTVVGGQQAGLFGGPLLVIYKAITVLIAARQAQAELGTPVVPVFWIAGEDHDFDEVDHIQVQRPELDVHKLS
IRRPWNLRTSVSRWQADEQAWEDAFRRLDESLPDSPHKTELLSRMRSSMNGLTTLSDTFAGWMAALFGDSGLVLLDADDP
ALRKLESPMFTRLIGANELLADALMRGRRELETLGYTPQVDVTDEQSHLFVIEPDGERVLLQRSGEGFADKKDLRAYSRE
ELLALAEREPERLSNGVLSRPLMQEFLLPVLGTVLGPAEVAYWAMTRDAFALHGMTMPILLPRLEFTLVEDAVTKWMDRY
DLSMDDVTEGIDERREAWLRAQSPFPVGERFEEARVRVKEAYTPLVDSLAAIEAGLPELGHTNLDKILHQIDYLERKAEQ
ALVRRHDAALSQWKRIELALMPLGKPQERVYNLCAYLNRYGDKWLSELLDADLPMDGGHYLVYI
>lcl|NZ_CP026520.1_prot_WP_042232439.1_4518 [locus_tag=PC41400_RS22790] [protein=MFS transporter] [protein_id=WP_042232439.1] [location=5027306..5028595] [gbkey=CDS]
MNVFHNRMFVKLFTASFFSQLGTMIGNMAFAFFLLDRFSSQPFYMTLAEMMYSLPTLFVFFIVGVAADRMNRKSICLYSN
YIRIGLTVLLFLSLFLEILPLTFLILFLRSAVAKFFAPAEIGLVQGILSKEQYMQAAGLNQTVFGLFMLFGTALGALAYH
WIGVEGAVIIDGLGFVLSALLIRSCSFEEKVQQPNGPGEWRDMSLKTVWGDFAEGMRYIWRKPLLLSMISGFILFGFVNG
GFAVLPLFTLKYKLAVDEYEKFASLYSVCLGVGVLVGSALGTIFAKKIKPHAMIIGSVVSTGLMVIGGSYINDVWLFLLG
LLLMGIFIAPANIAIGGWMPQIVEGKMMGRVNAWIDPLMMGAQSLALGLIALFYPRFVSLDWIYIVLGVLMLVVAVFYTL
KLPKQAAEEESAPKESQEKPVQGAASVSG
>lcl|NZ_CP026520.1_prot_WP_042232437.1_4519 [locus_tag=PC41400_RS22795] [protein=ATP-binding cassette domain-containing protein] [protein_id=WP_042232437.1] [location=complement(5028706..5029662)] [gbkey=CDS]
MSDQTILSIRNMKKHFDLGQGKILKAVDNFSIDIKRGETFGLVGESGCGKSTAGRTIIRLYEATDGEVIFNGENVHKLSG
RKMREFHRNMQMVFQDPYASLNPRMTVGNIIAEGLDIHGVASGSKRRERVIELLDAVGLNAEHASRFPHEFSGGQRQRIG
IARALAIEPQFIIADEPISALDVSVQAQVVNLFKKLQKDRGLTYLFIAHDLAMVKHISDRIGVMYLGNLVEVTTSKELYA
NPLHPYTESLLSAIPIPDPEIERKRERIILKGEVPSPLNPPSGCPFRTRCPKAMDQCAVTMPPMKEVEPGHFVSCHLY
>lcl|NZ_CP026520.1_prot_WP_042232435.1_4520 [locus_tag=PC41400_RS22800] [protein=ABC transporter ATP-binding protein] [protein_id=WP_042232435.1] [location=complement(5029655..5030701)] [gbkey=CDS]
MSKPKENILEVNDLKVSFNTYAGEVQAVRGVTFDLRKGEVLAIVGESGCGKSVTSQTLMRLIPSPPGVIKGGSILFDGKK
GKVDITKLSNKQMESVRGSEMGMIFQDPMTSLNPTMTVGRQISESLIKHQGMTKEQARQRSMELLRLVGLSNVEGRIKQY
PHEFSGGMRQRVMIAIALACNPKLLIADEPTTALDVTVQAQIIDLMRELSVKTGSSIILITHDLGVVAEMAERVVVMYAG
KIVEKGTVDEIFYDPRHPYTWGLLKSVPRLDTRADEELIPIPGTPPDLFAPPKGCAFAARCPYAMKVCVEEDPAHTQLSA
EHSAACWLLHPDAPKVERPVEVGGKQHV
>lcl|NZ_CP026520.1_prot_WP_042232433.1_4521 [locus_tag=PC41400_RS22805] [protein=ABC transporter permease] [protein_id=WP_042232433.1] [location=complement(5030716..5031651)] [gbkey=CDS]
MQNPGQSTVNASGKFAPLIKDEHLQEQIVRPSLTYWKDAWRRLKKNRLAMAGLFILILLTLVSIVAPFLSHFEYSATNPS
IANQKPNSVHWFGTDDLGRDLWTRVWWGTRISLFIGIVAAVIDLIVGVIYGGISGYYGGKVDSVMQRFIEIIYSIPFLLI
AILLMMVLKPGITSIIIAYSITGWVPMARLVRGQVLQLKEQEYVLASRTLGANTGRIILRHLIPNALGIIVVQITFVVPS
AIFAEAFLGFIGIGIRPPLASLGSLLSDGAASMRYYPFRLLFPTAVFSLILLSFNVLGDGLRDALDPKMRK
>lcl|NZ_CP026520.1_prot_WP_042232431.1_4522 [locus_tag=PC41400_RS22810] [protein=ABC transporter permease] [protein_id=WP_042232431.1] [location=complement(5031655..5032587)] [gbkey=CDS]
MFRYILQRLFYAIITLWLVITVTFVLMKLLPGDPFGEAVLKMSKENYQILLAQYGLDKPQWQQYLLYLKHVIQGDLGVSF
QFPARDVTSMISSGFKPSFELGIYSICVALVFGLLLGIIAALNHNKWQDRTAIFIAVVGVSIPSFVLGPLLAYYVGNKLG
LLPAALWKGPSYKVLPTLALSFGTLANVARMMRTSMLDVLNQDYIKTAKAKGLASHTVVWRHTIRNAIMPVVTIIGPVFA
GLITGTLVVEQVFSVPGLGKQLVNAIYSNDYTVISGMTIFYSAILIIVILITDILYGYIDPRIRLGKGGK
>lcl|NZ_CP026520.1_prot_WP_042232429.1_4523 [locus_tag=PC41400_RS22815] [protein=peptide ABC transporter substrate-binding protein] [protein_id=WP_042232429.1] [location=complement(5032691..5034367)] [gbkey=CDS]
MKAKKWVSTAVALTLVGSIAAGCGSDDKAATGDNGGTDKANAAKQELRINFNAEPPVLDSSKGTTNAAFTMLNALNEGLY
RLDKDGKPQPGLAAAKPEISKDGLVYTIKLRDAKWADGQPVKASDFVYSFQRTLDPSTKAQYNFMTAWIKGGNDIIKAKP
EEVEAKKKALGVKAIDEKTLEITLEKPVTFFEDLLAFPIFFPQREDLVTKNGEKYGGDADKIIGAGPFVLKQWNHDQNLV
FEKNPNYWDAANVKLEKLTVNIVTDSNSSLNLYETEETDVTNLSSDQFKAKSQTDKENIKVKNELTSMYFMYNQKNKVLA
NANIRKALTLAVDRDAYVLTLANNGTAAARGLVADGTQDGNKNEFRKTAGDTLPKADPAKAKEYLAAGLKELGIDKLPEL
KLLGDEQGSGKKSIEFLMGEWEKNLGYKAVGEPVPHKLRVERQNKGDYDIVLALWGADYNDPMTFLDMWVTGSEFNNVYY
SNPTYDEKIKAADKEADPAKRSQLLVDAEKILMEDMAVGPVYSRKVPYLVRNTVQGLVLPNYGMEWDVKGVTITEAKK
>lcl|NZ_CP026520.1_prot_WP_241688916.1_4524 [locus_tag=PC41400_RS22820] [protein=DUF3397 domain-containing protein] [protein_id=WP_241688916.1] [location=complement(5034786..5035184)] [gbkey=CDS]
MGLDSFIDGLSSIYAVLAMLPFLSFVLIWIIVYIVTKEKKKSTRITMDLTTFLLLGSVASLWNSLFSSSFGLWFLLLVIL
IVFGLIGGHQNDRKGKMELGKSLRIVWRLSFMGLSLLYVMFMVVRMGTAVLG
>lcl|NZ_CP026520.1_prot_WP_042232425.1_4525 [locus_tag=PC41400_RS22825] [protein=2-dehydropantoate 2-reductase] [protein_id=WP_042232425.1] [location=complement(5035187..5036149)] [gbkey=CDS]
MLVEVVGGGSLGLLFAAKLAATGCRVKLVTRSREQAEAVNEKGIRVEGWTELTAEVPALAFRDAVSGLNSADRSEARFIL
LAVKQTAFTKELASYLISSMENGAGLVSLQNGIGHWEKLDSEKARPFMIQAVTTEGARRLSHSEVAHTGRGWTRIGHVHA
KDTEPYIEKAPQINLKTVQDCLKQAGFDVFVSKSIERHIWSKLLINAVINPLTALLRVPNGKLPETSDSLLLMRKLFEEG
RDLALTQHVTVEESLWDEILEVCRKTAVNRSSMLQDIAAGRTTEIEAITGGLLGKAERAGMSMPTHETVYRLIKAAEQSG
>lcl|NZ_CP026520.1_prot_WP_042232423.1_4526 [locus_tag=PC41400_RS22830] [protein=RsfA family transcriptional regulator] [protein_id=WP_042232423.1] [location=complement(5036312..5036959)] [gbkey=CDS]
MTAIRQDAWSDEDDLMLAEVTLRHIREGSTQLAAFEEVGERIGRTAAACGFRWNSFVRKKYEAAISIAKSQRQKRAQLKK
HAAVSTISGGASLAVLDVLDPSGREVTEEALSVDAVIRFLRQWKQTYQDLMRHIKQMERSLNDKEEELSKYRRENEQLNK
QVNEVETDYRVVNDDYKALIQIMDRARKMAFLVEEEEESKPRFKMDANGNLERID
>lcl|NZ_CP026520.1_prot_WP_009671348.1_4527 [locus_tag=PC41400_RS22835] [protein=DUF2626 domain-containing protein] [protein_id=WP_009671348.1] [location=5037337..5037567] [gbkey=CDS]
MDRMFRVLGFWTLVIALMSLAGEMYPMSMLFFFQTVVFVLLGYLKLSERTYIVLFWGYMVISFTGFTYWSVFKMPL
>lcl|NZ_CP026520.1_prot_WP_042232420.1_4528 [locus_tag=PC41400_RS22840] [protein=SAM-dependent methyltransferase] [protein_id=WP_042232420.1] [location=complement(5037930..5039108)] [gbkey=CDS]
MSGFNGRRGIPSSAGETAADAGHPEIIAILKRKIEEEGPLPFRDYMTICLYHPEFGYYRREETKIGRDGDFYTSSNLGSV
MGEIIAGELLAYAVSVPAGRPVEFVEFGGGTGRLARHVLDALRSRSPERYASVSITAIESSAYHRALQRETLREHADRLR
FLTEAEWLAEPAGGGVFAVANELADAMPVHRVRCRDGRLELGAVGWDDRTGFTERWLPLGEGHPAAMHLGREGIRLAHGQ
QAEVNLGAGEWITRTAGRIRSGRLLLVDYGDTAGELYASHRMNGTLMCYRRHIADDNPYAAAGRKDITAHVDFTACMRAA
WAAGWTVARYETQREFLVRGGILGRLVEHAGRDPFSPEAKANRAIRQLLISDGMSELFKVLTLEREESRIRD
>lcl|NZ_CP026520.1_prot_WP_042232418.1_4529 [locus_tag=PC41400_RS22845] [protein=hypothetical protein] [protein_id=WP_042232418.1] [location=5039200..5040195] [gbkey=CDS]
MFKALISLSIAIAIPAVLISGCSERKAAKAAVKVALQKQTEMNAFAFAGQADLHLDGLPAGDSSTVPGQLFSFLKDSKLE
WKGAASLNPVRWESDLKAASRSGSAELTVPLLFQDNKLFFHMPLLNPDNEYFVTRLGLPPDSPASAADKTGTVPKAAASP
ALEHLSRLSSDLSKLLQEGADEAWFEHDGEVKLTDGSAARSYTASVTRMNAGAWNKKVQEKLPRFLEALKSNGLISEAQA
AEFGRSANSFKLSAPGKWTVIVDKEGFLRKLDMEFKFTGPGGNKSSASVQTFSFQQSFDAVNQTPEFRKPLPRQTKSIDE
VLKSLGSAGKQ
>lcl|NZ_CP026520.1_prot_WP_042232416.1_4530 [locus_tag=PC41400_RS22850] [protein=hypothetical protein] [protein_id=WP_042232416.1] [location=5040290..5041387] [gbkey=CDS]
MILKKFYLTFCALLIGTSLLFVPVHAENRTEAEKKETEALLQKGLTVHEIDREVARLAEEDVKLGGQLTDTEKKIAAQEI
VVKDAKKRAGKVIRSYYAGERESLWSVLLSVRSLPDLLKTYEYLDIILDHDHRSLTAHQDAYRELGAMKTKLANQRTDLQ
KTKENYLKQKERILALQSELNRELDASGRSELILALIQELNGRWEQEGLPLFRTYFRSLAGVMDYFPELLTPGGKDSHLT
LESGGGKFQMTDHQLNAFLRSKDGKFDDLTFQFKPDGVMAFGTKGDMDVSIKGTYEMEKDGDQPVIAFRVQELVFNGFKL
PDTTGKSLEKEFNLRIYPKKLFSFVSVEHITLEDGLLAIKLGLNW
>lcl|NZ_CP026520.1_prot_WP_042232415.1_4531 [locus_tag=PC41400_RS22855] [protein=hypothetical protein] [protein_id=WP_042232415.1] [location=5041526..5042050] [gbkey=CDS]
MQLKPALIVDFRQDGPAADSAPFDPELKETVELFDQDEEVFVFNREEELRQAKELVSRHGYEAETMNLVLLPPHAERTGH
FSDYGFESGAGHLYLYEPGIAVFRLGTGAVQDTDSPMKTASEAEPELAQRQMDEYVIARFHEADGLHFVVDRQHDELMDR
TARAYGCTVDFIIS
>lcl|NZ_CP026520.1_prot_WP_042232413.1_4532 [locus_tag=PC41400_RS22860] [protein=PhoH family protein] [protein_id=WP_042232413.1] [location=complement(5042130..5043461)] [gbkey=CDS]
MKKIFVLDTNVLLQDPNAMFSFEDNEVIIPAVVLEEIDSKKRNADEIGRNARMVSRLLDAMRTRGRLHDGITLDSGGSLK
VELNHKSFNKMQEVFAEVTNDNRIIAVALNYRTEEQTKSEPRPVILVSKDTLVRIKADVLGLTAQDYLSDRVVSDHEMYA
GCTTLHVHPSVIDEFYSFRFLSVPTLNLGYRLNPHEFVILRDELGTSKSALLKTSSDGKKLEPLFLSNEPVWGIAARNAQ
QRMALELLLNDDISFVTLSGKAGTGKTLLTLAAGLLKVEDERKYKKLTIARPVVPMGKDIGYLPGEKDEKLRPWMQPIYD
NLEFLFDTKKAGDIEKILAGLGSIQVEALTYIRGRSIPGQFIIIDEAQNLSKHEVKTIVSRVGEGSKIVLLGDPGQIDHP
YLDAASNGLTYAVERFKQEEISGHVTLEKGERSHLAQLAADLL
>lcl|NZ_CP026520.1_prot_WP_042232411.1_4533 [locus_tag=PC41400_RS22865] [protein=pyridoxamine 5'-phosphate oxidase family protein] [protein_id=WP_042232411.1] [location=complement(5043689..5044141)] [gbkey=CDS]
MAEEITALSEQLIAEFQKEKLVLLHTIDSETGAPTANAISWVYAPDPGRIRFAVDKRSRLIANLEKNPSATVTMIAAGTV
HAIYAEAAPAAEVLEGVPFKLACYDLNIRAVRDAMFYGARISVDPEFEKTYDKRAADKLDGQVFEAMKKA
>lcl|NZ_CP026520.1_prot_WP_042232409.1_4534 [locus_tag=PC41400_RS22870] [protein=hypothetical protein] [protein_id=WP_042232409.1] [location=complement(5044356..5044631)] [gbkey=CDS]
MTLQDALFNWLQIRIVADARPLDEAAGKTVGFFEEILREDHGVSEFEKTEADEYRLRIDYVRQGEKHSVFFDREHAERLL
ADIGSNPKYNE
>lcl|NZ_CP026520.1_prot_WP_042232407.1_4535 [locus_tag=PC41400_RS22875] [protein=YlaH-like family protein] [protein_id=WP_042232407.1] [location=complement(5044678..5044974)] [gbkey=CDS]
MYQWFAENFWLTCLIIFIMMSFVYNKVFRTRRLPVLKALIVYFLLALGSFMLTFFQVLGPQIPIVPSLAFAILLMLILRI
RYFIEGRRKKKDSGPAGQ
>lcl|NZ_CP026520.1_prot_WP_042232405.1_4536 [gene=typA] [locus_tag=PC41400_RS22880] [protein=translational GTPase TypA] [protein_id=WP_042232405.1] [location=complement(5045078..5046919)] [gbkey=CDS]
MFPREKIRNIAIIAHVDHGKTTLVDKLLQQSGTFRENEAVQERAMDSGDLERERGITILAKNTAINYKDHLINIMDTPGH
ADFGGEVERIMKMVDGVLLVVDAFEGTMPQTKFVLRKALEQNLTPVVVVNKIDRPNARPLEVIDEVLDLFIELGANDDQL
EFPVVYASALQGTSSLESDSQGENMLALYETIVSHIPHPTESVDEPLQFLVTLMDFNEYLGRIGIGRVNRGIIKQGQSVA
VMTREGGMKQARIEKLFGFQGLKRVEIEQAGAGDIIAISGIKDINIGETIADPANPEALPVLKIDEPTLQMTFIVNNSPF
AGREGKWVTSRKLRERLFNELETDVSLRVEETDSPDMFIVSGRGELHLGILIENMRREGFELQVSKPEVIIRHIDGQKME
PIERLLIDVPEDSMGSVMESLGTRKAEMVNMVNHGNGNVRLEFLIPARGLIGYRTNFLTLTRGYGIMNHAFDSYGPYAGA
GVGGRHEGVLVSMENGVATTYGLMSIEDRGTLFVTPATEVYEGMIVGEHTRDNDIVVNICKEKQLNNIRSAGKDDTVKLK
VPRIYSLEQALEYLNDDEYCEITPKSIRLRKKILNKNERERAEKQRKTADANA
>lcl|NZ_CP026520.1_prot_WP_042232402.1_4537 [locus_tag=PC41400_RS22885] [protein=hypothetical protein] [protein_id=WP_042232402.1] [location=complement(5047281..5047766)] [gbkey=CDS]
MSNKPLSMGIILVGVALVLLLGKIGFFTFIGALLWPLVLLVLPGLVLHALVFNRTLPSAAVFPGGVLVTYGILFFLCSLF
GWGLMSILWPGFIFGIAVGLYELVYFEKSRDNGLRMIAIGLALISVVLLLFNLVFTVSIYVFIVLLLAAGAVLIVSGKRI
W
>lcl|NZ_CP026520.1_prot_WP_241688720.1_4538 [locus_tag=PC41400_RS22890] [protein=hypothetical protein] [protein_id=WP_241688720.1] [location=complement(5047995..5048849)] [gbkey=CDS]
MMNYTRTVLTILAAALLTAACGKEEAPKPAAQSFGSATAAQTPADAAASPTAAAPPAGSAAPEATPAPTGTGAPGGQPPA
SGDAGGAAAPSASPGTAPRQSPAPGAAAAGGSAQPSSAPSGGTAAVTAPAASAPPQKNASAGTGAADKTPASAKPSAAPA
KSSSPVLKWTEFFDDEKQATPSNKFWDLNGKSVEIRGFMGEVLSFDKHWFLLIPAPGAECPFDNGDETYWNKIMIVFVPK
ETKLRYTSGPLKIKGRLDVGIKVDESGYKTMFRLYDASFESIKE
>lcl|NZ_CP026520.1_prot_WP_042229987.1_4539 [locus_tag=PC41400_RS22895] [protein=ABC transporter permease] [protein_id=WP_042229987.1] [location=complement(5048930..5050222)] [gbkey=CDS]
MNVFQVIWRNALHRKTLSILTMLSVALTAALLLFVLMWNEGVEKGAEKGYGPFEITMGAEGSKTQLAMSTYYHIGAPTGN
IPNAIYEQVKKEAEAEEVFAITTGDNLNGYPIVGVDAGYFAVRYGDKQLASGSLYGGLGEAVIGSHAARTLGLKVGDTFK
GGHGLVHSAEAAGEHAEEHHEKESGEHADASREGAQGDHNEHEGHDEHDGHEAFSYKVVGILPPLHSPDDRAVFTTLDYA
WAVHGEEAQHDREVTALMVKPKSLLGAQALKNKYDAMTGLQAMYTGKAVADVVNVVDKGTQVVQIVTYLCVLLAALTLLL
SLLAAASERRKDVALLRLIGKPGRYVWTALVGEGVFLTAGGLLFGLLLGHSAGAAASSLLFDFAGIQIDPWSLAPGELLL
AGGAVLLGIAASAIPAWQVYRVDPLYLFRS
>lcl|NZ_CP026520.1_prot_WP_042229988.1_4540 [locus_tag=PC41400_RS22900] [protein=ABC transporter ATP-binding protein] [protein_id=WP_042229988.1] [location=complement(5050219..5050995)] [gbkey=CDS]
MIKLTGVKKQFRLPGEGKLPILSVPEWEVEEGEQLALIGPSGSGKSTLLHLLCGVLAPDEGQIIVDDVPLHAQSQAQRDR
FRSEKVGIIFQDFHLMASLTARQNAELALPVKMPKSRKRELTAEWFERVGLADRMNHLPGQLSRGQQQRVAMIRALIRSP
KIVLADEPTGSLDRETAEDVMRLLLELTASANATLLVVTHDLELAAGFRTRTHIGELNEWYTPKGAAHPAGGSGRDPILP
GNMAEPLGTSGWKGEKLG
>lcl|NZ_CP026520.1_prot_WP_174446817.1_4541 [locus_tag=PC41400_RS22905] [protein=alkaline phosphatase] [protein_id=WP_174446817.1] [location=complement(5051079..5052788)] [gbkey=CDS]
MIAGTAIALLPLTLASGALAAGTTAAEKEQKAASKNVIVLIGDGMGPAQVSAGRTFSKNKLGKDHLELDSYFVGQATTYA
DRGEDGGTIVSGEVTDSASAGTAFATGNKTYNAAISVSNEDVSKPFASVIEASELAGKATGLVTTARITHATPAVYASHV
RNRDNESAIASQYLNDGNVDVLFGGGKQFFLSKEEKGKRTDKTIIPDFEKEGYKVVYDKQGLASLDGKTGKALGLFGNSH
VDYVLDRTNTVPNLAEMTSKAIDILSTNPKGFTMMVEGGRIDHAGHANDLPSVVQEMLDFDAAVKVAMEFAKKDGNTSVV
VTADHETGGLSLSRDNIYEINVDAWDDQQKSSELIGGELKNAKTIEDVKALVAKYTGHTDLTDDEAKFILAGDGSSYKQE
GAFNAVMSKRYLVGWSGHGHSAVDVGVWAYGPIAEKVKGQIDNTEIARAVASVAGIDLKAATDKLQSEYVYPKFKITREG
AVLYPARALAETLGAQVKWNGDSREVVLSAGGSNLSVNIDTQAAALNGQASGFQANIDNNVLYLPLEAFNKLTGKNLTWD
ALSERIKLK
>lcl|NZ_CP026520.1_prot_WP_042229993.1_4542 [locus_tag=PC41400_RS22910] [protein=MFS transporter] [protein_id=WP_042229993.1] [location=5052975..5054183] [gbkey=CDS]
MATNRKSKKKPLWTRDFAVTSLSNLLLFFSFQLLIPTIPLYAAGIGGGQTQIGLAMGTFTVASLLTRPFAGKALDTIGRR
HVLSAGLALCGLAVLGYSAAAVVAMLLALRFIHGIGWGISTTSLGTVISDIVPSERRGEGMGIYGLSNTMAMALAPLTGL
WIAAEYGFHPLFIFSTVLAFSALLLSLFIRSERKKPPASSGPFSWRGFRKGLIEPGALLPSLLLFCTAVCYGGIVTFLTL
FGKEASIGNVGWFFVCNAVMLLITRPLTGLVYDRKGPAWVLFPGAFFTGAGLWLLSYAHSASSLAAAAAVFGAGFGSVQP
ALLAWTIDRTAPSRRGAANSTFYSANDLGIGTGSMLLGAFASLVGSYGAAYRWSVLLIVLLIVIYSVYLLLTRRKSTQLK
AA
>lcl|NZ_CP026520.1_prot_WP_241688721.1_4543 [locus_tag=PC41400_RS22915] [protein=IucA/IucC family C-terminal-domain containing protein] [protein_id=WP_241688721.1] [location=complement(5054494..5055420)] [gbkey=CDS]
MVRWTQLITVLIFVKVVKTRHCRPGRLILQAVRDVQTENNHAWTEDPAKFREFYAKQFRVHLGGEKLLENTTGESDFLQE
GLEEIIGHYKDVYKTEHLRAIGATLSSYLSLIPAAFLHAMSHHNIALPLDPASTSIQYDGESFYLHLSGSPSTFVVPELP
REERRAHLVGSLFDSYMKPVFQRLRELTGIDKHSQWSYAALTLQRSYDNWMDKASTEELKYLLREDYEYVVGGSDPAWFE
RSGVRDRHPLDLPFKFVPDVYNEGKTVRMKYRCCLYYKLTGTYCYTCPAITEEHRSRRAEEIISARSK
>lcl|NZ_CP026520.1_prot_WP_009674003.1_4544 [locus_tag=PC41400_RS22920] [protein=TerC family protein] [protein_id=WP_009674003.1] [location=complement(5055521..5056192)] [gbkey=CDS]
MLEQLGLFLQIMLINIVLSGDNAVVIAMASKNLPAAQRRQAVWWGAFGAIALRLILTIIAVGVLKIPFIQAAGSVFLLWI
AVKLLMDNDAHSSVLQASTLSKAVWTIIAADFVMSLDNVLAIAAAAGNNLTILILGIGLSIPIIIWGSAVVVELLEKMPV
LIYMGAAVLGFTAGEMFLSDAKLSGWLLGEWPHWPVPWLGAVLVVGIGLLKKWLLPAKRKTVF
>lcl|NZ_CP026520.1_prot_WP_042229995.1_4545 [gene=thiI] [locus_tag=PC41400_RS22925] [protein=tRNA uracil 4-sulfurtransferase ThiI] [protein_id=WP_042229995.1] [location=complement(5056340..5057557)] [gbkey=CDS]
MMSPLKPDLLLVRFGEMMLKGKNRSRFEKKVVSQLRGVLAPFSQVKLVTEYGRVYIRLSDAPAEDVGKAVSKVFGIETYS
PVYTAPSDSEAIQAVAVALMKKVVKRPQTFKVEVKRVDKSFPFDTLEMNRIVGGAVLRACPELSVDVRSPETTLRVELRE
KGALLFVETFEGAGGYPLGTNGKAVLLLSGGIDSPVAGWQAMRRGLELEAVHFHSFPYTSERAKEKVIELARKLSAYSGG
SIKLHLVSFTEVQTSIYAAYKDNLLVTILRRAMYRIAETIAEREKALALVTGESLGQVASQTLPSLNAIGRAVSLPVLQP
LIATDKKEIIRMAEQIDTYTLSIQPYEDCCTLFLPPSPSTNPNLRVIEAIERGLPQLDEWILRAAQEAETVVVTPEDQAE
EDAYF
>lcl|NZ_CP026520.1_prot_WP_042229997.1_4546 [locus_tag=PC41400_RS22930] [protein=cysteine desulfurase family protein] [protein_id=WP_042229997.1] [location=complement(5057560..5058726)] [gbkey=CDS]
MLYLDNAATTQPYKEVAQTVADVMLQHFGNPSSIHKFGLDAEKLVKKSREVIASALRARSEEIIFTSGGTESSNLAIKGA
AYRYRSRGRHLITTAVEHASVTEAFRQLQAEGFEVTVLPVDETGQVRVEDVKAAVTGETILVSVMHVNNETGRIQPVGEI
GRWLREKQTVLFHVDAVQGVAKLPLSPDELGIDLMSASAHKFKGPKGAGFLYKRTGIDLQAQLVGGGQEQGIRSGTENVP
LIVGMAKALRIATDNLAEDIRRKRAFRRIAVEGILGLPELALTGSRDEEDMAPHIVHFTYPGMNSEVVVHALEQRGIYIS
TKSACSSGEPEPSKVLLSMGLDRAHASSGLRVSWTEEHSEADARRFVEALRGVIADLVPVSERTKGGR
>lcl|NZ_CP026520.1_prot_WP_042230000.1_4547 [locus_tag=PC41400_RS30630] [protein=lytic transglycosylase domain-containing protein] [protein_id=WP_042230000.1] [location=complement(5058957..5059742)] [gbkey=CDS]
MVSIDPRLMKELLKLEMLNKSTLFSGAGGGVSGASEPGEEFSSLLNTLLMEQGGLSADPAGESALGILPLEALQGAWTGS
AGAVPAAAGTGLSVQRQAKALRSYETAAGAVQASLPASAREAAYDAYIVQASERTGVDPALIKAVIRQESSFNPNTVSRA
GAKGLMQLMDGTANGLGVTNAFDPEQNIQGGAQYLSYQLKRFDGNVGVALAAYNAGPGRVNKLGIKNDSDLLEKMHLLPN
ETQQYVKKVMNAKERYDREFT
>lcl|NZ_CP026520.1_prot_WP_042230002.1_4548 [locus_tag=PC41400_RS22940] [protein=ABC transporter permease] [protein_id=WP_042230002.1] [location=complement(5059867..5060640)] [gbkey=CDS]
MKFLSLVQNEIMKITSKKSTWIFYIFLLLLSLFVGLPIKLWIPGFSDGHNYISFAGTVFMVCSLFVTLFALVLGAQTVTE
EYKDGTIKQLLIRPAGRTAILLSKYVACVIMVLIAYAVLFLSSVAVGAALFGTAELPNEHFIVLSSSYWYSLPDMLFMFT
LAFFIATVFRSSALAITVAIVMNLAGSAFAALQYNWARYLIFNNTNWKIYDTDPLINQGTPPPFPGMTFGFSLTIYVLHL
VILLGVTILVFRKRDVA
>lcl|NZ_CP026520.1_prot_WP_042230005.1_4549 [locus_tag=PC41400_RS22945] [protein=ABC transporter ATP-binding protein] [protein_id=WP_042230005.1] [location=complement(5060637..5061560)] [gbkey=CDS]
METEQAMELVGVSKTIGGKPIIKDLSFSVRRGEICGFLGPNGSGKTTTIRMMVGLMSMSAGEIRIAGFPVRTRRSEALAH
VGAIVENPELYPYMSGHKNLVHFARMAAGPVPAGRMEEIVELVGLTDAIHDKVKTYSLGMRQRLGIAQALLHRPTVLILD
EPTNGLDPGGIRELRDYLRNLAKTESIAILISSHLLAEVELICDRALIIQEGRLVGERIVRGGATAEDELVLVEFELKGI
EAGEHTAGEYGLLRRENGLYTANLSKSAIPEIVSRLVGQGIDVYQVAIRKPTLEDTFLQLTKGEDSR
>lcl|NZ_CP026520.1_prot_WP_042230007.1_4550 [locus_tag=PC41400_RS22950] [protein=DUF4190 domain-containing protein] [protein_id=WP_042230007.1] [location=complement(5061702..5062118)] [gbkey=CDS]
MTHHKNRKKQGRTREHSQKLDPAEANSSAGREHAGRSDREEYAGELAPGLAGRNVEDAGARSREDADSDAWISARALGIM
ALVSSVLSFFLLPFVLGPLGAILGYMALADGSRRAGGWAIAIGLISFFSNLFFVPFVI
>lcl|NZ_CP026520.1_prot_WP_042230010.1_4551 [locus_tag=PC41400_RS22955] [protein=DUF1540 domain-containing protein] [protein_id=WP_042230010.1] [location=complement(5062164..5062373)] [gbkey=CDS]
MPEVKCAVSNCAYWGQGNNCRADIIMIEVDEHADADLSSEFAGEEFDTRHKDQAADRANTCCHTFKEKK
>lcl|NZ_CP026520.1_prot_WP_042230013.1_4552 [locus_tag=PC41400_RS22960] [protein=YpuI family protein] [protein_id=WP_042230013.1] [location=complement(5062543..5063043)] [gbkey=CDS]
MAMDNVKKACETTRKKLKEAADRIESFLNTYSLPQLNAESDPEMEDFYRGYLQDTRHLLVFCEVAYEKLGVSLRRPTFNV
DFTEKVLYEVYHNCISPFFYPKNECYSEDGRYAYTGQDAIRFRKKPNREVRDLTIELSKIFEELREELSYYENDYITQRR
MQGERV
>lcl|NZ_CP026520.1_prot_WP_053228709.1_4553 [locus_tag=PC41400_RS30120] [protein=S8 family peptidase] [protein_id=WP_053228709.1] [location=5063238..5065331] [gbkey=CDS]
MRRPTAALGGMILGSLLVVTSACQTKPEARPVQPEAELKYKQAVLDQDVAVTDQLCRNQCLLHVNQMYASLNHADPKKIK
ETMREMQSKHGHIISLNWIPVKTGSGTPGSSPSAGAPAAPGGTTVPSGPGAGAAGSSGNAGVPGASGASSAPSGAGTSPS
ASSGEIVSGKLSKELAKSAQPYIEEAKKAASQGKTYQSPNFTVADKPYFVLNVPAPDKKSSVVAVVRQHILQEVSNHQRK
NLRLQSFHSDKSRWKIESVDTDTLRSKRVTHPDDHEGTSHYERHDVVVQFNRPPSEAELSRIRTQIGATKVKLIGNTYII
TARSLEAKQLMDYFRKRQDVRFAEPHYIYETNDLSRGTKPLSWFFDWWPGSRSDNSGRTRSGAGEQAAEQPDVIPNDVLF
NKYQWNLPITNTLRGWNFNKGTQDVKVAIVDTGADVNHEDLKGQLIEGYNVLDPTAQPMDDVGHGTHVAGIIGAQVNNGL
GVAGMTWYNKIMPVKALDSSGVGSTYAVAQGIIWATDHGAKVINLSLGNYASSDFLHDAIKYAYDRDVVLIAASGNDNTE
RPGYPAAYPEVFAVAATDSSNGRASFSNYGDYIDVAAPGVSIASTYPQNQYAALSGTSMACPHVTALAALLRSANPSLKN
TEVMELMRQTATDIGDKGRDTSFGYGLINVDKAVEAALGRGPVQAGLPGTKPAPVAP
>lcl|NZ_CP026520.1_prot_WP_042230018.1_4554 [locus_tag=PC41400_RS22975] [protein=Nif3-like dinuclear metal center hexameric protein] [protein_id=WP_042230018.1] [location=complement(5065947..5067062)] [gbkey=CDS]
MFAKGQTVIQMFEKLAPKHYAVPDDKIGLQLGTLQKEIKKILVALDVTDEVVDEAIRTGTDLIIAHHAIIFRPLAHLQTD
TPAGRLYEKLIKNDIAVYIAHTNLDVAEGGINDLMAEALGLKDLVPLDEVHTEKLSKLVVFVPESHHEKVLQALFQAGAG
WIGDYSHCSFNISGTGTFMPREGTKPFLGEQGKLERADEVRVETIVPAGIQRAVVQALLKAHPYEEVAYDLYPMDLKGRT
FGLGRAGKLPEPLTLRELVEAAKQAFDVPTVRVVGDLDRTVRKAAVLGGSGGRYVRSAMFAGADVLVTGDIDYHTAHDAL
AAGMCLIDVGHNVEKIMKRAVADYMNRALAEKKYETVAAASEPSTEPFQFV
>lcl|NZ_CP026520.1_prot_WP_042230020.1_4555 [locus_tag=PC41400_RS22980] [protein=class I SAM-dependent methyltransferase] [protein_id=WP_042230020.1] [location=complement(5067038..5067796)] [gbkey=CDS]
MVKLSQRLLEIAKKVPHGAKLADIGSDHALLPTFLVQQGTVPSAVAGEVNRGPAEAAQRQVRDANLSGQIEVRLGDGLSV
LRAGEVDAVTIAGMGGALIASILNAGEDKLTGVSTLILQPNVGEDIVRKWLRSHGWHLAGETILEEDGKIYEILTAVRMK
PEDEGDAGLYADRLLDGLRVTGETLLQMGPYLIQNPPPVWYAKWEYELDKLAMIRRQLDRSVSGNAEEKRSGLEAEISRI
KEVLECLRKDKP
>lcl|NZ_CP026520.1_prot_WP_042230023.1_4556 [locus_tag=PC41400_RS22985] [protein=hypothetical protein] [protein_id=WP_042230023.1] [location=complement(5067800..5068627)] [gbkey=CDS]
MDDEKRTAIIKEIEHWRRSKLLPAQYCDFLLNLYMETGADHSRRVLGISTKSIRGSHWKFWVAAVVIIIVLSLTLLNFNS
FGISLQIGVSAVFVVSWYIIGFRQKEEQPFRAQLSTGIASASLIGFGVWILYGNGLTAQLYIGSFVALCGLVWLLVGLLG
RTPLLQYCGWLALLITYGFVLSQRLEPATWYKLQLAWLPLSFILMWIGWLIQHRWKKPGLVLLLAGATVWIAPELFGLAW
GTSAQQLLQLSLAGKVAAAGIALFAYRKNWTEWVA
>lcl|NZ_CP026520.1_prot_WP_042230026.1_4557 [locus_tag=PC41400_RS22990] [protein=alpha/beta hydrolase] [protein_id=WP_042230026.1] [location=complement(5068710..5069696)] [gbkey=CDS]
MSKTLLIGLCILAVLAVALTAAAFYMFGLGIRRSSKEFLQNSPDLAHHIEEIAVDPSPASADNWLARQTLEQVSIQSADG
LRLNGWYLEAERPSDVTALLAHGYSGQGRDMASFAQIHHELGYNVLMPDNRGHGQSEGEYIGFGWTDRLDYVNWIRYILQ
RSGENARILLHGVSMGGATVLMASGERLPDQVKGIIADCSYTSVKDILSYQLKRMFKLPAFPLIPLTSLICKLKAGYFFG
EASALRQVKRTEKPILFIHGEADTFVPFFMVHGLYKAAPAGKDLFMVPRAGHGLAFMTDKDGYRSRVTDFAKQCLEEKQK
VLQARSEA
>lcl|NZ_CP026520.1_prot_WP_042230028.1_4558 [gene=rpoD] [locus_tag=PC41400_RS22995] [protein=RNA polymerase sigma factor RpoD] [protein_id=WP_042230028.1] [location=complement(5069756..5070880)] [gbkey=CDS]
MANDQHTEIETELTLEQVKEQLIELGKKRSSLTYKDITERLAPYDQDPEQIDEFFEQLSEMGIDVGNESDEDEGRMRREG
DQENDEFSFEDDLTLPPGIKINDPVRMYLKEIGRVPLLSAENEVDLAKRIEQGDEEAKRRLAEANLRLVVSIAKRYVGRG
MLFLDLIQEGNMGLIKAVEKFDHTKGYKFSTYATWWIRQAITRAIADQARTIRIPVHMVETINKLIRVSRQLLQELGREP
TPEEIAKEMELSTDKVREIMKIAQEPVSLETPIGEEDDSHLGDFIEDQEALAPADAAAYELLKEQLEDVLDTLTEREENV
LRLRFGLDDGRTRTLEEVGKVFGVTRERIRQIEAKALRKLRHPSRSKRLKDFLE
>lcl|NZ_CP026520.1_prot_WP_042230030.1_4559 [gene=dnaG] [locus_tag=PC41400_RS23000] [protein=DNA primase] [protein_id=WP_042230030.1] [location=complement(5070920..5072746)] [gbkey=CDS]
MGYGRIPDEVIEAVLQRHDIADVIGKYVHLTKQGHYLKGLCPFHSEKSPSFTVTPEKQIYHCFGCGAGGNMFQFLREIEG
YTFAEAVRHLAEEADIPVTWEESLSADNPQQKDRSKMLEANEKAAKLYRYILMNTDQGKNALDYVRNRQMSDKLIDTFQI
GYAPAMWDTLVQQLQKAGYDLPLMERGGLVSARSEGDGYIDKFRDRVIFPIWDAQGKVIAFGGRALGDVQPKYLNSPETM
LFNKSRVLYNFHQARPQIRKTNTMVLFEGYMDVVKAWEAGVRNGVGTMGTALTPEHAELIRRNAGRIVICYDGDSAGQNA
AYKAIPLLEQAGCEVKVAMLPDAKDPDEYITAYGSERFVREIIDYAVPSIKYRLHYIRRHFRLQEEGDRIRYQQTALKMI
ADLSSIEREHYLKQLAGEFETSYESLKQDLANLRMKSEKNRTVRDNIDVPWNNGRHNRKETPKAPLYPAYHNAERQLLAV
MMHDRDVSAYVEERLGDEFNVDAHAVLAAYLYAFYAQNSTPSVSRYMAMIQDDKLENLASSIAMMGAHHGVNEIVIDDFI
KEIKKYPKQQEISRKKEEMIRAERSGDPLRAAQILSEIIALEKQLKFS
>lcl|NZ_CP026520.1_prot_WP_009671649.1_4560 [locus_tag=PC41400_RS23005] [protein=YaiI/YqxD family protein] [protein_id=WP_009671649.1] [location=complement(5072808..5073251)] [gbkey=CDS]
MDADACPVKPEIIRAAAQYSVHVLMVASFDHRLKGQEGIEVIQVDRSDQSVDLFIANHIRRGDILVTQDFGLAAIGLGKG
AVCLSNRGQEYTDRTIDFLLDRRHTQGKLRRGGKHSKGPKPFTDEDREFFLHGLTKVLLRLQENRPV
>lcl|NZ_CP026520.1_prot_WP_042230031.1_4561 [locus_tag=PC41400_RS23010] [protein=pyruvate, water dikinase regulatory protein] [protein_id=WP_042230031.1] [location=complement(5073355..5074182)] [gbkey=CDS]
MNPSNNPVIVYVVSDAIGETGESVVNAVALQFYPAPIVIERVPFVHDIEEIDRLLAKVARQQGIIAFTLVIPELRDYLTQ
RAEQAGFAYVDLLGPLIQTFEKAFNRKARHQPGGNHPLDEDYFKKMEAIEFAVKYDDGRDFSGIVKADIVLVGVSRTSKT
PLSMYLAHKKYKVANVPLVPEMQPPEVLFKVPKNKVIGLLIDPDKLNAIRRERLRTLGLASNATYAKTERIKQELEFAGT
VLSRIGCHVIDVSNRAVEETASLVLEQIRRHPHLS
>lcl|NZ_CP026520.1_prot_WP_042230032.1_4562 [gene=recO] [locus_tag=PC41400_RS23015] [protein=DNA repair protein RecO] [protein_id=WP_042230032.1] [location=complement(5074411..5075157)] [gbkey=CDS]
MLHRVQGIVLRSVDYGEGNKIITLYTRELGKVGVVARGVKKVKSRLGAVTQLFTYGDYVFYRQGKLGTLNHGEIIEPHHK
LREDLYKSAYASYLVEMVDRMLGEEDRSLFLFDQLEAGLMAIEQEKDMQIVAHVFELKMLAASGYTPVLDKCASCGRTEE
LAAFSAAAGGAMCSRCKHLDPYAAAVTESTLKLMRLLQQVDLSRIGNVNVKPSTKSQLKHSMRLYMDAHVDVRWKARSFL
DQMDKYNI
>lcl|NZ_CP026520.1_prot_WP_009671644.1_4563 [locus_tag=PC41400_RS23020] [protein=YqzL family protein] [protein_id=WP_009671644.1] [location=complement(5075207..5075353)] [gbkey=CDS]
MRDFTWHYFWSTGEVDAYLLYKDLASDPELAGEELMQQEQEAVHSVEA
>lcl|NZ_CP026520.1_prot_WP_042230034.1_4564 [gene=era] [locus_tag=PC41400_RS23025] [protein=GTPase Era] [protein_id=WP_042230034.1] [location=complement(5075464..5076363)] [gbkey=CDS]
MNKEKFKSGFVSIIGRPNVGKSTLMNQVIGQKIAIMSDKPQTTRNKIHGVYTTDNSQIVFLDTPGIHKPSSKLGDYMMKV
AHSTLGEVDAILFLADVSEGIGGGDRYIIEQLKNVKTPVILVLNKIDQVHPEELLPIITTYKDLYEFAEIVPVSALLGNN
VNTMLEQVVKYLPEGPQYYPADQITDHPEQFVCAELVREKILHMTREEIPHSIAVAIEDMKVEKNGVVHISAVIYVERDS
QKGIIIGKQGSLLKEIGKQARHDIEALLGSKTFLELWVKVKKDWRNQERVLRDLGFRNE
>lcl|NZ_CP026520.1_prot_WP_042230035.1_4565 [locus_tag=PC41400_RS23030] [protein=cytidine deaminase] [protein_id=WP_042230035.1] [location=complement(5076381..5076785)] [gbkey=CDS]
MKPEELLEHARQARLKAYIPYSNFGVGAALLDKDGGVHHGCNIENAAYGPTNCAERTAVFRAVADGHKPGSFQSIAVIGD
TEGPISPCGVCRQVLIELCGPDMPVILGNLNGDYTVTTTGELLPGAFTSKDLQY
>lcl|NZ_CP026520.1_prot_WP_042230039.1_4566 [locus_tag=PC41400_RS23035] [protein=diacylglycerol kinase family protein] [protein_id=WP_042230039.1] [location=complement(5076849..5077550)] [gbkey=CDS]
MSSFQRFLRGFRFAYEGIKYAFDTQRNMKFHFVVAFLVFLAAVILGLPHWDVLFLLLAVVLVIMTELINTAVEKAVDLAM
PELHPLAKIAKDTAAGAVLVAALFAVVVGMVVFYGPADRMLRKAQEAAAANMPGMVWTLIALVVLVTIVIETRFSDRGKL
VRPSLLAAVLAALATLIAVIAGQTIVTLLSGTLAALALMVLAERKHRELSSLLLGSVIGAAVTLLAWLWRGWG
>lcl|NZ_CP026520.1_prot_WP_042230040.1_4567 [gene=ybeY] [locus_tag=PC41400_RS23040] [protein=rRNA maturation RNase YbeY] [protein_id=WP_042230040.1] [location=complement(5077547..5078083)] [gbkey=CDS]
MALILAWNDEQDILPISPKFAEKLEELLKLAGEEEGVEDGEVALTFVDDETIHELNRTYRNIDRPTDVLSFAMLEQGEDE
PAILYGEEEPDEDSAESDAHVDGNGEESFEEPLGDIVISVPRAAAQAEEYGHSVERELGFLFVHGFLHLIGYDHDNEEAE
QEMFAKQEQILQKAGLTR
>lcl|NZ_CP026520.1_prot_WP_042230042.1_4568 [locus_tag=PC41400_RS23045] [protein=HDIG domain-containing protein] [protein_id=WP_042230042.1] [location=complement(5078117..5080333)] [gbkey=CDS]
MKKEVAVHDMQTNTKLSGWKTSFPVRLLLLFALMLIICMTLFSRLIPQTFDIKLGAVAEKNIYAPFQIENSVATEKAKED
AAKKLQPIYKIVNLRNEAMIDAIYDRILQINADPDVKFEDKVSVYRTVIPAIILDTEKTAIRTIGENNFGNETFIDEINS
KVSDQKYRIPEEIYYKFPRLTKEDVAAMAPITREIVSKLMSDQISDAQTARAKVAELVNSSDLSKNTLRELVQEIARVVI
TPNKFLDQKATDDARGEVRENVKPVYINQGDIIVNEGDVITDEIYKRLKAAELLKEKANYWPHVGLVLLGLLFVAMIYLF
VRQSTLPIRTNNSQLLMFVLIFAINMIGMKVVALGQNPEYPFIGYLAPAAIGSMLITILLNPQLAFIASVLFSIMSSIIF
NMHGTQLFDYRYGLVTLVTCFASVFVIQKASQRSSILKAGILISFIGMITIVSLMLLDNLTRQDMLYSLSFSAASGILTA
VFVIGLLPFFETAFGILSPLKLVELSNPNHPLLRKLLTETPGTYHHSIMVGNLAESAAESIGADGLLCRVGSYYHDIGKT
KRPSYFIENQTNIENPHDRIDPALSKSIITAHARDGVEMLKEYNIPKPIRDIAEQHHGTTLLTFFYHKARKLAEEEGREV
REEDFRYEGPKAQCKEAAVVGIADCVEAAVRSLRNPTIEQIDTMVRKIIKSRLDDGQFNDCDLTLKELDTIAKTLNESLL
GIFHSRIEYPSELPAKAK
>lcl|NZ_CP026520.1_prot_WP_042230044.1_4569 [locus_tag=PC41400_RS23050] [protein=PhoH family protein] [protein_id=WP_042230044.1] [location=complement(5080376..5081347)] [gbkey=CDS]
MGENAKTIKVALADTSVAQAVFGPQDRFLHLIEQETNARIFTRDAELTIQGDAEEVRKLEHLFEVLLELVRNQYPLSERD
VLYAVELAKQMKADQLLELFKGEITTTHKGKPIRVKTIGQKQYVSTIQKKDIVFGIGPAGTGKTYLAVVLAVTALKEGKV
KRIVLTRPAVEAGESLGFLPGDLQEKVDPYLRPLYDALNDVLGPEQVVKSLERGLIEIAPLAYMRGRTLDDSFIILDEAQ
NTTPEQMKMFLTRLGFGSKMVVTGDVTQIDLPRGKTSGLIEAQRILNGIEEIGFITFAEQDVVRHSLVQRIIVAYNEDSL
ISG
>lcl|NZ_CP026520.1_prot_WP_042230047.1_4570 [gene=yqfD] [locus_tag=PC41400_RS23055] [protein=sporulation protein YqfD] [protein_id=WP_042230047.1] [location=complement(5081354..5082544)] [gbkey=CDS]
MQSAVLKWIQGYVKIEVKGERAAEFINRALSKGLSLWDIRVSGDKLVELCLPCRDALRLRPLLKETGCRMHPLKREGMPF
AARKLAKRKTFLAGGVAFIASLFILSSFVWSIKVEGNVKIGTQTILDAARKEGLYLHQWKYRMGDSEKLSFGLQTHLPGT
SWVGVEIKGTQVTIKIVEADKPEEKPLVSPRNLVASKHAVVTEIQAKRGRPLVQPNAYVRKGDVLISGTLGDEANQKIVV
ADGYVKGMVWYTSKIEVPIIQTYETYTGESKTRNYLVFGSKGLQVSGYGKLPFEHYQTIPEQKMLQWRSFKLPVGWLSEK
LMASETVERKLDPAEAKAIGLERARADLMSEAGKDARIVTEKILHEKSENGKVYMEVHFEVEEFIMEEQPIVQQGE
>lcl|NZ_CP026520.1_prot_WP_042230050.1_4571 [gene=yqfC] [locus_tag=PC41400_RS23060] [protein=sporulation protein YqfC] [protein_id=WP_042230050.1] [location=complement(5082570..5082848)] [gbkey=CDS]
MRRLTQKWKQMTANMLDLPQDVVQDIPRITMIGNVQLYVENHCGVLHFSSEMLRLQLTRGKLEVSGKELVIRAILTEEVL
IEGIIADIRYMP
>lcl|NZ_CP026520.1_prot_WP_129112528.1_4572 [locus_tag=PC41400_RS23065] [protein=hypothetical protein] [protein_id=WP_129112528.1] [location=complement(5083181..5083672)] [gbkey=CDS]
MMERLIELIANNFFFVIMIGIFLVSMLSGKRKKTGNPNRMPDFSGQGLPRRPAASTRSASGSDREPDRRGDWSAEDAAAD
WTSAEPAAAGRGRSGGFAASDARGSSAAPADTPRKRETRAVREETAVSGGSADGVDLHPRQAVQGVIWAEILGPPRAKKP
FRR
>lcl|NZ_CP026520.1_prot_WP_042230054.1_4573 [gene=floA] [locus_tag=PC41400_RS23070] [protein=flotillin-like protein FloA] [protein_id=WP_042230054.1] [location=complement(5083817..5084800)] [gbkey=CDS]
MESYFYLLIAAVVLILLMVFFTFVPVMLWISALASGVRISILTLVAMRLRRVIPSRIVNPLIKATKAGLGLNINQLESHY
LAGGNVDRVVNALIAAHRADIQLPFERAAAIDLAGRDVLQAVQMSVNPRVIETPTVSAVAKDGIEVKVIARVTVRANIDR
LVGGAGEETIIARVGEGIVTTVGSSGSHKDVLENPDMISRTVLGKGLDAGTAFEILSIDIADVDVGKNIGAHLQTEQAEA
DKRIAQAKAEERRAMAVAQEQEMKARVVEMRAKVVEAESEVPQAMADAMRNGKLGVMDYMNYKNIDADTHMRSSIGKMSE
EGKEVKE
>lcl|NZ_CP026520.1_prot_WP_042230057.1_4574 [locus_tag=PC41400_RS23075] [protein=NfeD family protein] [protein_id=WP_042230057.1] [location=complement(5084852..5085622)] [gbkey=CDS]
MISLRRTFLGCATVLGSIGCLPGLAAASAGTAAQQGFTSALGGFLTHPAVAAILLLIGIVGIGLELLFWTFGLLGSIGVI
GFGLYFLGNYLAGGAGSGDIGLFVLGVLLLLLELVIPSFGILGIAGSISLFSGVILAADNPQTAALLLVIAFVAAAVLLF
IAVKKFPARGVWNRFILKEELTTEQGFVSSSFKLHLMGQTGTSLTPLRPSGTAQFGEDRVDVVTEGGFIPAGRQVKVVLV
EGSRVVVHEEEAGAEK
>lcl|NZ_CP026520.1_prot_WP_042230058.1_4575 [locus_tag=PC41400_RS23080] [protein=GatB/YqeY domain-containing protein] [protein_id=WP_042230058.1] [location=complement(5085833..5086273)] [gbkey=CDS]
MSLSERLNDDMKQAMKSQDKFTLSVIRMVKASIKNIEIDQRRTLDDNEVLDVLNREVKQRKDSLQEFKNAGRDDLADSLH
AEIDILMKYLPQQLTEEEVSAIVKQTIQEVGASSKADMGKVMTALMPKVKGRADGKLVNQLVQQNL
>lcl|NZ_CP026520.1_prot_WP_024094532.1_4576 [gene=rpsU] [locus_tag=PC41400_RS23085] [protein=30S ribosomal protein S21] [protein_id=WP_024094532.1] [location=complement(5086290..5086463)] [gbkey=CDS]
MSETRVRKNETIDAALRRFKRSIAKDGVLAEIKKRKHYEKPSVKRKKKSEAARKRKF
>lcl|NZ_CP026520.1_prot_WP_009675046.1_4577 [locus_tag=PC41400_RS23090] [protein=histidine triad nucleotide-binding protein] [protein_id=WP_009675046.1] [location=complement(5086586..5086933)] [gbkey=CDS]
MDCLFCKIRDGQIPSKQVYEDEQVLAFHDIQPAAPVHILIIPKKHIASMKDAGEEDWALIGSVHKAAQQIARDFGIEESG
YRLINNCGPDSGQVVFHIHYHLLGGTNLGPLVAQR
>lcl|NZ_CP026520.1_prot_WP_042230061.1_4578 [locus_tag=PC41400_RS23095] [protein=hypothetical protein] [protein_id=WP_042230061.1] [location=complement(5087108..5087410)] [gbkey=CDS]
MQKHEAKSLCQQHKYRYVLIQLTDGTVHDGIIENVDDDYVYLAIPVGGRAGGESETRAFGWGYPYYRYPNHGYPYYPRRR
FYRGIFPLVALTVISLLPYY
>lcl|NZ_CP026520.1_prot_WP_241688917.1_4579 [gene=rsgA] [locus_tag=PC41400_RS23100] [protein=ribosome small subunit-dependent GTPase A] [protein_id=WP_241688917.1] [location=complement(5087543..5088565)] [gbkey=CDS]
MSAEGLTAGRVALEHKRMYRVKTDSGEVLAEVSGSMRYQARSRGDYPAVGDWVALSVREEEGRATIHGVLPRKSMFVRKV
AGNVTEEQIVAANVDVIFLVNALNADYNLRRLERYLVMAWESGASPVIILSKADLCASPEEVALRVAETEAIAPGVPVHA
VSSLLGEGVEGLRAAYLGIGRTAAVAGSSGAGKSTLINALLGHEAMDTGAIREDDARGRHTTTHRELLLLPGGGVLIDTP
GMRELQLWEADSGLIQSFDDVETLAASCRFPDCRHSSEPGCAVRGAIEDGTLDQARYDSYVKLQKELAFLARKESKSLQT
AEKDKWKKLHKEMRRKPSQP
>lcl|NZ_CP026520.1_prot_WP_042230066.1_4580 [locus_tag=PC41400_RS23105] [protein=hypothetical protein] [protein_id=WP_042230066.1] [location=5089193..5090440] [gbkey=CDS]
MKKILAPSLASIFLFVCSSNVVLADYMTCDNDPPSSDTYCQGSAGENWEKYTNKPSSFNSDHRFYKGGQTFNRSRYNWTV
FPLGGNSTGEYSVGFYLNSASFTGNANYYIEGMNIGGLDQNTAPVGWNNFPYKTHPWSNGKQAHVAVEPLSINSSVGADG
VRVYYSTNTIVSTQENTVESENLKLDAASIKEKMLNAIDYYNYVRGSFLYDFSNSNQKYTIDFEVDQSKNLGSNIVLKDS
STKENLFRVTSSSNEIIQVDEKNKKYSKSSKTAERTKILAPRISKDKGPDGEEANQYMYRNDPANASAASTVTFPQTIAF
WLDDELKNYKITGQEDLLNRKATVIEGKLSKELSEKLLSTSFKLWVDTETGILLKLNTFNVDGVVINKIVVNDISFDYKK
TIFDTSNNKGYVPQQ
>lcl|NZ_CP026520.1_prot_WP_053228710.1_4581 [locus_tag=PC41400_RS23110] [protein=HAMP domain-containing sensor histidine kinase] [protein_id=WP_053228710.1] [location=complement(5090970..5092376)] [gbkey=CDS]
MSSIRSKVVWNFLLIIVFIVLLLGGLFLGTIYQYYYGSALQALSERAKFTVDYNNRYNSLSPLQERARNILKELAKDEIS
RVEVVDLEGNLIRDSYGFTSKRKIDTSDVQSALLGEPGKWIGLNTDTKERIMALSYPLKNGDRVVGVLRFTTSIEQVDRM
VMRTAWIIIGIGVAVILVSLAVSLLLANRIVRPIREVTDVAQSMAQGDFHNKAVKRHDDEVGHLADTLNYMADEILRNDR
LKNEFISSISHELRTPLTSIKGWSETLVSGDLNEKEETLTGLSVISKETDRLIGLVEDLLDFSKLQSGNMKMHREPVDVN
RLVKDIYQQFAGNCQRRRMELQISLAENSLYVFGDQNRLKQVLINLLDNAMKFTQENGVIRIMTRVEGSLVQLIVEDEGE
GISPEAVKYVTEKFYKADNRQPGSGLGLSICQEIAQLHGGTLQVTSELGQGTTVTVELPRMPEEEKVL
>lcl|NZ_CP026520.1_prot_WP_009675036.1_4582 [locus_tag=PC41400_RS23115] [protein=response regulator transcription factor] [protein_id=WP_009675036.1] [location=complement(5092357..5093073)] [gbkey=CDS]
MRVLILEDEESIRGFIRINLKRNDIDVTEAATGEEALQLVKDEGPFHIAILDVMLPTQLSGFDVCAQLRKMYPKMGIIML
TAKSQDTDKAEGLEIGADDYVTKPFSPVELIARIKALYRRLEIPEQESAKQSLSSGIFTLQLKERKLFKGRTEIDITPTE
FAILKLFMEQPDQGISRDEILDAVWGKHFVGDLKIVDVNIRRIRQKIEVDPSHPAHIETVWGYGYLWKKESALEQHKE
>lcl|NZ_CP026520.1_prot_WP_042230070.1_4583 [locus_tag=PC41400_RS23120] [protein=hypothetical protein] [protein_id=WP_042230070.1] [location=complement(5093094..5094503)] [gbkey=CDS]
MKRRIKWIPRPAAALLTAGLSMLLLSGCQLISSSPAALLKAPAMSSSLYQLKEAVTHYLPPGAVLNTPLKLPDNSAAGPI
YQVDLNGDNQKEAVAFYRLQKNEFELGALILEQVNEEWRKVTDIHELGREIDRVDFVDVTGDGAPEMLVGWSAGEAVNLN
KELFVYSMKGSVVSQMEKIPYTEIAIGDLTGDGVPEVAALLLDRDKLDASASLYSFKNGEKKLMQKLPMDGGVNGYYQVV
IGKATDSQNGIFVDAGLGANTSYTSLLVWNNGKLKDVFSAKDEQGNVQTFKEYKGRSQDVNGDGIMEIELLKIPPFIENS
NPTDMPWIHQWYQWDGANGLKWVHENYYDNNVTYRFDFPESWRGKITLERSKSDRADGYTTFYYLSANRKEKAELLTIRY
YPTEQWENMKNELQKKEIDFVELDTSYGRKTVAFLPEKPTNLSVKSLQEYNQMKLGLKDIQERLKPVYY
>lcl|NZ_CP026520.1_prot_WP_241688722.1_4584 [locus_tag=PC41400_RS23125] [protein=polysaccharide deacetylase family protein] [protein_id=WP_241688722.1] [location=complement(5094679..5095623)] [gbkey=CDS]
MKKRKVFRLRYVLMIALGLFIAYNVSHALLTKETATGPSSGETAVTPSPSPVPTANEPEKKPEKSARPDPTPTPDPTRKP
DPTPDPTATPDPTPTPQTTKAPQGGGTANTASPIYSISTKNKLVALTFDDGPDLHWTPQVLEVLSKYNAKATFFLVGRQV
DKYPDMVKRIVDEGHAVGNHTWSHAQLPKLTPEQAREQMTSTDEALKKVLGKGTYLFRAPYGAVNDNVKAVAAQQGQLII
GWSIDTKDWAGSTPDQIMNNVKQHIHPGSIILQHSFGGKGGDLSNTVEATRRICEYLQAQGYQFVTVPELLAAK
>lcl|NZ_CP026520.1_prot_WP_042230074.1_4585 [gene=pflA] [locus_tag=PC41400_RS23130] [protein=pyruvate formate-lyase-activating protein] [protein_id=WP_042230074.1] [location=complement(5095951..5096709)] [gbkey=CDS]
MKGRIHSIETFGTVDGPGIRFVLFMQGCALQCGYCHNPDTWDTCGGRTVTVDEILAEMEPYLPYYRKSGGGITVTGGEPT
LQAAFVADLFKACKERFGVNTALDSSGFCEVGHEHVRELFEETDLVLLDLKMMDRDKHIALTSQPNDKILRTARWLSDHG
RPMWIRRVLVPGMTDAEDELERLGSFLAGLATVEKFELLPYHKLGTYKWDLLHREYPLKDARTPTEEEVARAYELIERAR
SQNVSQPTCQSL
>lcl|NZ_CP026520.1_prot_WP_042230076.1_4586 [gene=pflB] [locus_tag=PC41400_RS23135] [protein=formate C-acetyltransferase] [protein_id=WP_042230076.1] [location=complement(5096749..5099019)] [gbkey=CDS]
MAIEEREMRMKSAEEGVYRGFRKGKWQHGIDVRDFIELNLTPYEGDEAFLAPPTEDTKALWEVVKDLTRREREQGGVLDV
DVNTVSTIVSHAPGYLDKDKERVVGLQTDEPFKRSVQPFGGIRMMEDACEAYGFKLPQDMVRLFTDIRKTHNQGVFDAYT
SDMRAARKAGIITGLPDAYGRGRIIGDYRRVALYGIDALVAEKKKDLLLLEVDAITEDVIRAREELSEQIRALGELKKMA
ADHGFDITGPASDAREAFQWLYFAYLAAIKEQNGAAMSLGRVSSFLDIYIERDLREGILTETQAQELVDHFVMKLRIVKF
LRTPDYNELFSGDPTWVTESIGGMSLSGRTRVTKNSMRFLHTLYNLGPAPEPNLTVLWSEQLPEAFKKYCAKVSIETSSI
QYENDDLMRPFYGDDYGIACCVSAMRIGKQMQFFGARANLAKALLYAINGGVDEKLGVQIGPESVLITSDVLDYDEVMHK
FKGVMEWLAKLYMNTLNVIHYMHDKYCYERIEMALHDREIVRTMACGIAGLSVVADSLSAIKYAKVRPIRNEKGIAVDFE
IEGDYPMYGNNDDRVDSIAVSLVEAFMGMIRKHKPYRDAVPTQSVLTITSNVVYGKKTGSTPDGRKAGEPFAPGANPMHG
RDRKGALASLGSVAKLPYEHSLDGISNTFSIVPKALGKEPQTRKDNLVHMLDGYFESGAHHLNVNVFDRQQLLDAMEHPE
QYPQLTIRVSGYAVNFIKLTREQQLDVINRTFHGAM
>lcl|NZ_CP026520.1_prot_WP_042230078.1_4587 [gene=adhE] [locus_tag=PC41400_RS23140] [protein=bifunctional acetaldehyde-CoA/alcohol dehydrogenase] [protein_id=WP_042230078.1] [location=complement(5099223..5101832)] [gbkey=CDS]
MAVQENETVKPAAAAAKEQVDRLVQNAKKAQAAFEGLDQSQVDRIVQEMALAGVDRHMYLAKLAVEETGRGVYEDKIIKN
LFATEYIHNSVKYDKTVGVIDEDPTTGLTSIAEPVGIIAGVTPVTNPTSTTMFKALIAMKTRNPIIFAFHPSAQKCSSEA
ARVLYEAAVSHGAPKDCIQWVEKPSMEATHELMHHEGVALILATGGSGMVKSAYSAGKPALGVGPGNVPCFIERTADLRQ
AVTDLILSKTFDNGMICASEQAVILDEAIAAEAKAMLKAQGCHFLSAKETKKLSQFGMNAGCAVNAAIVGQPAARIAEMA
GFEVPAGTKILIAELDGVGPAYPLSAEKLSPVLACYTAASAQAGIARAAEIAAFGGLGHTSVIHSDDEGVIAAFAKELPT
CRIIVNAPSTHGAIGDIYNANIPSLTLGCGSYGRNSTSSNVTAVNLINVKKVAHRTVNMQWFKVPEKIYFERNATQYLAK
MPDITRVLIVTDPMMVKLGYVERVEHYLRQRRMPVAIEVFSDVEPDPSVDTVERGTDMMNAFQPDCIVALGGGSPMDAAK
AMWLFYEYPDVTFHSLKQKFMDIRKRTFKYPKLGTKAKFVAIPTTSGTGSEVTSFAVITDKKNGHTKYPLADYELTPDVA
IVDPEFVYSLPKTAVADTGMDVLTHAIEAYVSVMANDYTDGLCLQAIKLVFRNLEQSHATADKLAREKMHNASTIAGMAF
ANAFLGINHSLAHKWGGQYGTAHGRTNAILMPYVIRYNASKPSKYASFPKYSHYCADERYAEIARFLGLPAATTEEGVKS
LIDAIRELNQKLGIPASFKELGFDADDFESRVEYLADKAFEDQCTTANPKMPLVSELADVYRDAFYGRI
>lcl|NZ_CP026520.1_prot_WP_042230081.1_4588 [locus_tag=PC41400_RS23145] [protein=hemerythrin domain-containing protein] [protein_id=WP_042230081.1] [location=5101998..5102525] [gbkey=CDS]
MSTITVERLPYAAMDPGSQLSDALYYLKQQHEFLEADLLELQMLARGIGAAGSRDSLRTDLNQLKVRAERFLKFFQQYQQ
DEQETLFPIVDLYTGDKMGPTAVMKQEYNMAVHNLRLFFQAFKEDSHTLDPLCAKRSVSHVLQSCFILLEHMRKEKEWVF
PLAEQMLTDIEQFYS
>lcl|NZ_CP026520.1_prot_WP_042230083.1_4589 [locus_tag=PC41400_RS23150] [protein=Crp/Fnr family transcriptional regulator] [protein_id=WP_042230083.1] [location=5102737..5103462] [gbkey=CDS]
MNTGTNPVYYTNPRNTAAFKTENLQKLTDIMHEQKLQPGAYLFWEGDTADKLYFIKNGRVRATKSTDEGKDLVLYLYQQG
DLIGMIDPYDDTKHWYSAEVLEETLIGVAHKKDVEMLIWQSGEFALEFMRWMGLSHRMTQTKYRDMMLYGKPGALCSTII
RLSNTYGRPEEDGTIRITKKLTHSELSDMIGSTRESVNRMFSDLRKAQAIEMDNGYIVIRDLNYLRGICHCENCPVEVCR
M
>lcl|NZ_CP026520.1_prot_WP_042230085.1_4590 [locus_tag=PC41400_RS23155] [protein=formate/nitrite transporter family protein] [protein_id=WP_042230085.1] [location=complement(5103764..5104558)] [gbkey=CDS]
MFTETVEKMAAAAAQKKNFLSESLLRYLTAAGLAGAYVGLGIVLIFSVGAPLFAAKSPVTTTVMGLSFGVALTLVIFAGS
ELFTGNNMYFTIGALSRKVTWKDTLVNWFWCYLGNLTGAMVLVLLVVGAGVFANIKPDHLLMTIAAKKMNMDTTQLFFRG
ILCNWFVCLAIWTSMRAKEDTAKLILIFWMLYGFIASGYEHSVANMTVLGLALVLPHPETVTLAGWFHNMIPVTLGNIVG
GAVFVGLLYYFISPAKRADGKLKA
>lcl|NZ_CP026520.1_prot_WP_042230087.1_4591 [gene=nirD] [locus_tag=PC41400_RS23160] [protein=nitrite reductase small subunit NirD] [protein_id=WP_042230087.1] [location=complement(5104670..5105005)] [gbkey=CDS]
MNQQLQEKPETYVCIGSQGDFPARLGKTVFVGEEEIAVFHLTDGRLFALQNRSPHRKGGPLCEGMVSGDLLYDPLYDWKI
NLLDGKVQAPDTGQVKTYPVKLDGGHVWIGL
>lcl|NZ_CP026520.1_prot_WP_042230090.1_4592 [gene=nirB] [locus_tag=PC41400_RS23165] [protein=nitrite reductase large subunit NirB] [protein_id=WP_042230090.1] [location=complement(5105062..5107497)] [gbkey=CDS]
MNRRKLVLIGNGMAGVGVIEQILKINTGLFDITIVGSEPHPNYNRIQLSYVLEGSKTTDEIVLNDWNWYRENGIKLLTGT
TATALDTTAKTVTTDRGDVLPYDALILATGSKPFILPVPGADKEGVVGFRDIADCEAMTEAARTYRKAAVIGGGLLGLEA
AKGLSKLGMEVTVVHLMDDLMERQLDPAASGMLRAELERQGLRFAMGKQTKELTGGVRVEGLAFTDGTVLETDFVVMAAG
IKPNSALGAAGGLEVNRGIVVDDFMRTSAPDVYAVGECCEHRGVCYGLVAPLFEQGSVLAKHLCGTQTAPYEGTVLATKL
KISGVDVFSAGEFMDQPDLTIVKMQDDWRGVYKKVLVRGNRIVGGVLFGDVGQATQLQQWIRSGREMTEDVHAALMGTPA
GEESGGGAELLADDDIVCGCNGVTKGTIVEAVRTQGLKTVDEVKACTGACRSCGGCKPLVEQILAGVLGDGYEAGTAAHG
ICGCTQLGRDEIVAAMRERKLTTTRETMAALNWSNPEGCSKCRPALNYYLTMLWPETYVDEKESRFVNERLHANINKDGT
YTVVPRMYGGVSSAEELKRIGDVAVKYNVRAVKMTGGQRIDLMGVRKEDLPKVWEELGMPSGYAYAKALRTVKTCVGSTF
CRFGTQDSMGMGQLLEKKFERIDMPAKFKMAVNGCPRNCAESCTKDVGIVGNDGGWEIYIGGNGGIKPRLADLAVKVKTD
EELLEITSSLIQFYRETANYLERTSDWVERLGLAYIQNHAITDEANRRELVNRMETALTRAHDPWQKVIGDRQLRQELYE
GVNVPVGPPTE
>lcl|NZ_CP026520.1_prot_WP_042230092.1_4593 [locus_tag=PC41400_RS23170] [protein=SMC family ATPase] [protein_id=WP_042230092.1] [location=complement(5107830..5111444)] [gbkey=CDS]
MRPIRLQMSGLQSYRESQEIDFTQLCDAGVFGIFGPTGSGKSSILDALTLALYGKVERASGGTQGIMNHAENTLAVSFTF
ELSHPTGAVRYRVDRQFKRGSDVSVTNTVSRFVQLSGEEPVVLADKSTDVTNRVHQVLGLSMQDFTRAVVLPQGKFAEFL
SLTGKDRRQMLQRLFHLEAYGDHLSARVSGRLKEADGLLKQIAAEQQGLGDASEEALAQAGERLKEAAEYAVVQRKLFQQ
ASSRYEEMKQIVSVQKDERQTELELRKTEEEEPGIRQREDNLKRAEEAEKLRPYVEHAETSAKAEAEHRGRIAQARDRSS
QAEVKLDTALQASERARQELAGSEEPLLIRLEHLNQALEQERQLADLHKQCEGLKTSEEQLLSQLSAVKEAFRKERETKE
KALGKQAELKEMLKSVEVGYEQRSLLQEAWDAKRELDMQLRQLQELRQETGGRRRTMEAGERELAVRAGELTAWTEQFAR
WLGELGAGRTEALQLRQRLAKQEEAAAAVLQIRKREDRERELRNLAEKLAHELHEGQPCPVCGAVEHPAPYREEEEAEKA
SPGAEECETLLRAIREAGMETDRHVFGLDSLAKRALEHERAGTDQASNPAAKSASGDETGHAAVEAALMLAEAAAAVGAD
LELPPDKLTIGAADLTDDKDAIAAASAEAAESLAATAGIRQGEAGISEGEALRNLAAFEGPQAETARVGRLAAELEAGFQ
ALLRRLKPLETGMQAAEAELRSQRAALQADETKAQALEQSIRGREDAWRKRFGARGLELDTVEARLKQLGEQERQAEELR
GRIEKSIPFIDGVFAKIEQLRTEAAELDKTALQLATKLQGLSQLAAEKAQQLHERVGSEPVPRQIAAAQTELDALRTRAE
QTRLAHEQARREQQEAARALSAAEQAAASAAGQLAEAQERLDRALKASSFADADAVRSAYVAEAQRAEWSAEARRHRERE
QTLRSRLAQLREQLAGRTVGEEDWTGMLHELREAQERDEAALAVRAKAERDCEELQAKHEQWRRLEERRSAQQTLHGRLA
KLQSVLKANAFVEFLAEEQLMQVSRAASQRLGQLTRQRYAIEVDSGGGFVIRDDGSGGVKRPVGTLSGGETFLTSLALAL
ALSAQIQLKGEVPLEFFFLDEGFGTLDQDLLDMVVTSLEKLHMDRLAVGVISHVPELRARLPRKLIVEPAEPSGRGSRVR
LELL
>lcl|NZ_CP026520.1_prot_WP_042230095.1_4594 [locus_tag=PC41400_RS23175] [protein=exonuclease SbcCD subunit D] [protein_id=WP_042230095.1] [location=complement(5111460..5112647)] [gbkey=CDS]
MRILHTGDWHLGRTLEGRSRFEEQVAFVDELVRIVREGNIDLVLLAGDVYDTVNPPAAAEQLFYDALSRLAEDGRREVVV
ISGNHDHPDRLGASSPLAKAQGISLIGLPKAGLMEIGIPRTGETAKLFALPYPSESRLGELLSDDAGEEVLRRAYSERVG
ELVKAQTAGFRPDTVNLIMSHIYVLGGKESESERPIQVGGAYTVDTSALDVGAQYVALGHLHRPQTLKAQSPIRYCGSPL
AYSFSEAGQAKSLTVLDVKPGEPASIEEIYLTSGRPLVNWKAKEGLAEVHRWLDEGRDANAWIDLEIHIAEAMSLHDIQT
LRKAHSGFIHIKPVYPQQQPAEEEAAALSSAGLPVDELFRRFYERQTGGARPEPELVQLFMELLQDHSDGPEQAE
>lcl|NZ_CP026520.1_prot_WP_042230097.1_4595 [gene=addA] [locus_tag=PC41400_RS23180] [protein=helicase-exonuclease AddAB subunit AddA] [protein_id=WP_042230097.1] [location=complement(5112728..5117320)] [gbkey=CDS]
MTTTPRTKPPGTTWTDEQWDAIALRGQNMLVAAAAGSGKTAVLVERIIRRISDEASPLDVDRLLVATFTKAAAAEMKQRI
REALEKELTANPDSQHLKRQLALMGRASVTTLHSFCLEVIRRHYSLIRLDPGFRIANETEAELMRHDQLEEMMEQYYGES
GEDSPFWRLVDTFGGERSDVPLHLLIQKLYDESRSHPWPEHWLREMALMFGPKNYELITGSDSAGEDEMTAALAEVSEGY
AEAAAASDSAVVGTAAETASANAAEPAINPADTLADTGGADIALTAENADKAAEAGDEGEGAPAEFWLKGLTEDVRLELR
GTAELLKQALEITGQPGGPLPYATAVQDDLTMVEGLTDIAENYPWDILFSAFQSVSFGKLKPCKKDEVDPQLQDQVKQLR
SRAKERITSLQEELFGRPPERFLEETREMAPVIHALVDLVIDFGGRYSKAKAAKGLVDFADLEHFCLRILRSEESAPDRL
IPSRAALEYREQFAEVLLDEYQDTNRVQEAIVELIAEPLPGNRFMVGDVKQSIYRFRLAEPGLFLEKYKSFTGDGSGPGR
RIDLARNFRSRAEVVDAVNFIFRQIMNEGVGDIAYDERAELVNGASYPAAEGQNLSVEMMLIDRAAAGQASDGEETDAEE
SAGEISAEEQEGELGGILEGQELETAQMEARAIARKIQAMLDPQDGRPYLVYDKRSGSMRPATYRDMVILLRATQQWSPI
FIEELRQLGIPAYADLNTGYFTATEVEVMLALLKVIDNPYQDVPLAGVLRSPIVGLTADELASLRISGKNIPFYEAVRAY
AQGIRPLGSTEGGSDLQEAGEWKPDEELRGKIADFLRRLEEWRSEARQGALARLIWKVYGDTGYFDFVGAMPGGQQRQAN
LRALYDRARQYENTSLRGLFRFLRFIERMKDTGGDLGTARALGEQEDVVRIMSIHKSKGLEFPVVFAAGLGKMFNQRDLL
DPFLIHKELGFGPRFVDTKLRVGYPTLPQLAIRRRMRIETLAEELRVLYVALTRAREKLFLLGTVRSLDKQLQGWSRHLE
SDNWPLPDDEVARARSYLDWLGPALMRHRDASLWREKAGLQEHSPPALAADGSKWSFEVIAPEELAAMGKADESVTIQAD
RLEALKHLNKVPTFTEEWDVALAKSLSWTYPFAQATAYFSKTSVSELKRQADRSTASDHEEIPSAELMPSPGSGLILPTR
LLKMAGAAEASRLNAEHSPGTPASGRDTEGTRSGGNPVFDLTTFMDSPAFVQAKELRGREAAETQDEQEPQETRDAEDTQ
KAENVQDPEVGSQGTQDAGDTQEAQAAQGTQAARGPIEPPGEAGGEVPSGAAVPGTPSAGSSARTSLLRRPRFMEEKKLT
PAERGTVFHAVMQHMPLKEVTPQAVEAVIRDMIDKELLTPAQARAVDTEILAGFFRTQIGLKLAAATRVWREVPFSLGLP
AGRVYPNADASTRGETVLVQGVIDCLFESDGNLIMLDYKTDRVKGVGLEALRERYKIQLGLYGQAVEKIWKRKLAGTYLF
FFDGGHVVEI
>lcl|NZ_CP026520.1_prot_WP_042230100.1_4596 [gene=addB] [locus_tag=PC41400_RS23185] [protein=helicase-exonuclease AddAB subunit AddB] [protein_id=WP_042230100.1] [location=complement(5117317..5120901)] [gbkey=CDS]
MPVRFITGRAGSGKSKLCMDEIIENLKQRPEGPPLILIVPEQATFQTELALVDHPDVGGTIRAQVLSFRRLAWRVMQERG
GTARQPIDDVGKKLLLHKLLHKRKGDLRFYRSVYEQTGFVDRLNDLFAEWRRYCVTGESLREHVDRYDDFLGSKKQLLRD
KMRDLQLIYGAFEHEISLRYLDGEGTLAKLAEELGQSSLVRDAQIWIDGFHGFTPQELMVVKGLFEHGAGTTVSLCVNRP
YYAGEIPDELDLFHPTARTMAQLQELVDAVGAGPAETVAVQPVLSPRFAASPMLAHLENSFEQRIGRYGSPYIPAGPGER
EQLSLYAAVNRRAEAEGVAREMTRLAREQGVRWRDMAVMVRNMADYEDLLETAFTDYGIPHFFDQKRTVLHNPLAEFIRS
ALETVLHNWGYDAVFRCVKTDLLLPWDGDAEEVGAKTLTEQRAAMDRLENYVLSYGIQGNRWTDGKPWTHRSRASLEDEE
TAANEAEERYLQEIQQTREWVAEPLLRFQKNLKRAKMVQSRTQALYELLEDVHAPERLEQWYAEAAANGRPEKAREHSQI
WGSLMDLLDQLVELMGEEEISAELYGELLETGLESIKLGLVPPALDQVLVGSMDRTRSMKIRYAFILGVSDGVIPASPTE
DGVLAEAERELLLQTGLPMADGNRRRLLDESFLIYSSLAVPSRRLSLSYPLADEEGKSLLPSELIRQVKGLFPELREVQW
MGEPSPLMSAEEQLSYMQNPSRAMSYLGVQLKQWMRGAAIGDLWWGTYNYFAGEPEWKGSMNRLVQGLLYANREDRLPRE
TSRLLYGRHLRASVSRMERYVACPFAHFLTHGLKLKERRVFRLEAPDIGQLFHGALNLFVRGLQEDKVDWASLSTEETYR
RTSDVVDRLAPRLQGEILLSSKRYRYISHKLKQVVGKAAVMLGEHAKRSLFSPVGLEIDFGPGGMLPSLTFQLDNGCTME
IIGRIDRVDQARGDSGLLLRVIDYKSSPTALDLSSVYYGLSLQMLTYLDVILTHAPQWLGMSATPAGVLYFHVHNPLLQF
TNGMTVQDVEKELRKRYKMKGLLMADPEVVKMMDSDLAGATGHSQLVPAALSKDGGFYKSSSVASEEQWDTLRGYVRDRI
RDIGTEITEGRVDITPYRTGNKTACQHCSYKAVCQFDPALDGNEPKHLAKIGRDRVWAELTRKETGSGKEEQDR
>lcl|NZ_CP026520.1_prot_WP_042230104.1_4597 [locus_tag=PC41400_RS23190] [protein=spore germination protein] [protein_id=WP_042230104.1] [location=complement(5121024..5121248)] [gbkey=CDS]
MVGSIIGNIKINSVSNSSNVQIGNAAFIVMSSNNKMSGGAASFSPGDSFGGISNNPNSITNEPEINQGTRALNG
>lcl|NZ_CP026520.1_prot_WP_042230106.1_4598 [locus_tag=PC41400_RS23195] [protein=GNAT family N-acetyltransferase] [protein_id=WP_042230106.1] [location=complement(5121465..5121938)] [gbkey=CDS]
MLWNDSDYKWRDFHAQKIKSQTDQEKIRELLGYAVFPDPESVEEAVRFYGQAPGRELYGYESEGDLIGLVGFSTDAEGTL
TIHHLVVDPECRGAGYGRGQILELIQLKKPREIVAETDEDAVDFYRSIGFTVSSLGEKYPGTERFVCRYEAEPKEEE
>lcl|NZ_CP026520.1_prot_WP_042230108.1_4599 [locus_tag=PC41400_RS23200] [protein=AEC family transporter] [protein_id=WP_042230108.1] [location=complement(5121962..5122885)] [gbkey=CDS]
MTYFIHIFLNNIVPLAVMILIGAAMFRAFRLDIKTLSKLNFYVFAPALVFVKLYESELSVAILGQVLLFFLLFFTAISVM
TELFVRIRRYKGGMKGAMRNSIIFYNSANYAIPLNQMVFVGDALTLSVQMIVMMMQNLLPNTLGVYNVNAHKSSWKEILK
TVAYLPTIYMIPLAFVLRHYQIAVPQPLSIPLHYIADGFMATALLTLGVQLGGMKWAFRFSDVLWSNFLRLCMGPLIGLG
VVWLLGLHGVMAQALVLSCAVPTSLSSVLLAVEYDNEPEFSSQAVFMSTVLRMFTVTLVIYLLPYIG
>lcl|NZ_CP026520.1_prot_WP_042230110.1_4600 [locus_tag=PC41400_RS23205] [protein=hypothetical protein] [protein_id=WP_042230110.1] [location=5123008..5123361] [gbkey=CDS]
MKRIVVLIFAAFFAFSANASAAYTAVSQDLYVSQAAPHKASIGLWSTTNEVVQITVYSKSPNGQLSPLYSKNVAIGPAVA
SPVEFSVGYLSPGTYVIKGEFPGSYGSLGPVSLYPSF
>lcl|NZ_CP026520.1_prot_WP_042230113.1_4601 [locus_tag=PC41400_RS23210] [protein=hypothetical protein] [protein_id=WP_042230113.1] [location=complement(5123429..5123704)] [gbkey=CDS]
MGKRKNADSESLDKLDYRVKVLLKEWAGLSGFYGYKSMSLDEILELVNTDGWETKVDKSDPSVRRVVMVKEDTVRKAKFF
LDEKGRALKAD
>lcl|NZ_CP026520.1_prot_WP_129112529.1_4602 [locus_tag=PC41400_RS23215] [protein=NucA/NucB deoxyribonuclease domain-containing protein] [protein_id=WP_129112529.1] [location=complement(5123759..5124208)] [gbkey=CDS]
MNKKRKKNKKTASIFLFLLIVLVSVVAKSLGLFDGETGKSTSADQTIYFPSDKYPETAKHIKDAIAKGASSVCTIDRKGA
EENRRQSLAGVATKKNYDRDEWPMAMCAEGGKGADIAYIKPSDNRGAGSWIGNQLDKLPDGTRVEIVVK
>lcl|NZ_CP026520.1_prot_WP_042230117.1_4603 [locus_tag=PC41400_RS23220] [protein=DinB family protein] [protein_id=WP_042230117.1] [location=complement(5124332..5124865)] [gbkey=CDS]
MVTFKDIVPYMDNVRRQLTEVLNSFGPDEKQLREKPDGWNVTEVVEHLAKLEGMILYQLKQVVGQTPAGPSEPLDEKATD
VMGILQASGIIGRKIEAPDPARPTGASSYEEALQMLDEVRAATKALIPKLAERNTNDLLFPHPAGFEMNAAQWAHFIAIH
ETVHVNQLKRIRAANRR
>lcl|NZ_CP026520.1_prot_WP_042230119.1_4604 [locus_tag=PC41400_RS23225] [protein=class I SAM-dependent rRNA methyltransferase] [protein_id=WP_042230119.1] [location=complement(5124966..5126330)] [gbkey=CDS]
MAAIVLQKKRKKRLEQGHPWIFASEIERVEGEPVPGQVVEVRTHAGQPLASAYYNPKSQITGRVVSYKPLEAMTKEFFVE
RFGRCARHRARFLHDSDSYRLVYGEADFLPGLTVDKFADVLVVQILTLGMDVCREAITAALVEVFAPQGIYERSDVPVRE
LEGLTQTTGALYGECPRYVRILENGLHIEVDIEQGQKTGYFFDQRENRAATEPLMKGWGAESGIAVREVDHEGRMQRLPV
NANGKVVTFPYWDGATVLECFSHTGSFTLHACKYGAKKVTCLDISEHAVESARTNVKLNGFEDRVEFVVADAFEYLRRQV
KGVEERRQRSLAGQGAKVDTSKPVGEGRTWDVVILDPPAFAKTRSAVEGACRGYKDINLHGMKLVNEGGYLVTASCSYHM
APELFLETVHAAAVDAGKILRLVEFRAAGKDHPRLLGVDEGNYLKFAIFEVRSR
>lcl|NZ_CP026520.1_prot_WP_042230121.1_4605 [locus_tag=PC41400_RS23230] [protein=Na/Pi symporter] [protein_id=WP_042230121.1] [location=5126493..5127761] [gbkey=CDS]
MWLSLALPLLAGLSVFLFGMKMMELALNQLAGPHIYRFLETATRTPAHGLVTGTALTALLQSSSAVTVISIGLVNAGVLT
FPRTLGIILGTNIGTCLTTELIGLNLGAAALPLLLASSAVWLASWMPGPGRRQAGAGGAAVAGNRPGGSASGDAKTGPAG
SPAPALRSGGTREGRPAGSARTDGGLAAGSRPAPDRAWPPVRSETPGSSHAHDVVRPDSRPAGDQGPAQEPPAGRGWLHA
VRYAALAAAGFALVLIGIRIMQSIGPALQHQGLFAWFLEQARVSLLWGVIAGAAVAALIHSSSAVIVMAMGFAASGVISP
ELGIAVTLGSNVGTCVTAWLASIGGTRAGVFVAWSHILLNVGGTVLFFPLIGLLQEVSSWMSADPSAQIARAQTLFNVAS
SLIALPFCYLPFSRKKNLQPRT
>lcl|NZ_CP026520.1_prot_WP_042230124.1_4606 [gene=deoC] [locus_tag=PC41400_RS23235] [protein=deoxyribose-phosphate aldolase] [protein_id=WP_042230124.1] [location=complement(5127727..5128404)] [gbkey=CDS]
MTGISDLASYIDHTLLKPEAQSSAIRKLCGEAKEFGFYSVCVNSGWVPLCREELQGTAVKVCAVVGFPLGASLPEVKAFE
AAKAAESGAEEIDMVLAVGSLLEGNAEAVRGDIRQVVDAVKGRAIVKVILETGFLSDEQIVLACRLAEEAGAAFVKTSTG
FGPGGATAEHVRLMRQSVSAAVRVKASGGVRDLETARSMIEAGAQRLGMSSGVALMSGAAGSSSY
>lcl|NZ_CP026520.1_prot_WP_042230126.1_4607 [locus_tag=PC41400_RS23240] [protein=NUDIX domain-containing protein] [protein_id=WP_042230126.1] [location=complement(5128401..5128844)] [gbkey=CDS]
MVPMKEISGGGVVFRKQEERTEVQLIQDRFGKMTLPKGKMEPGETIEETALREIAEETGIAGRIVEPLDVISYRYEHPEH
GTVEKEVHYYLVEAVGGDLQAQIEEISGVEWMSPETAWKKQAESGYDNNDGVVLAALQKLGYEVKLV
>lcl|NZ_CP026520.1_prot_WP_042230129.1_4608 [locus_tag=PC41400_RS23245] [protein=peptidoglycan-binding protein] [protein_id=WP_042230129.1] [location=5129121..5130098] [gbkey=CDS]
MLGKKSMIAVTCTGLMFSSLMIANSASATETCGGYLSNMTDPLAIRMENGEKVIWDNVDPVLIQKRDKYVSLLKQKNWTI
TFNSAYRPYQYQRHLYEIVQNASKSSCVAAEKAAHGLGTEVAKPSNTAPHTRGIAFDAIVKDSAGNALNGRTFVNSQLKD
VAAQAGLSFPLPTTDGVHHIVAGGSTTPDDGVLRLGDQGPEVTELQQQLNKVGYTVSTDGIFGSGTDTAVKQFQSAHALV
SDGIVGVATNAKLDTLSTTTTVLRSGSSGDEVKVLQRLLTKKGYTVTADGIFGSGTDTAVKKFQTANSLTSDGIVGSATW
SKLRQ
>lcl|NZ_CP026520.1_prot_WP_042230131.1_4609 [locus_tag=PC41400_RS23250] [protein=PBP1A family penicillin-binding protein] [protein_id=WP_042230131.1] [location=complement(5130291..5132750)] [gbkey=CDS]
MADTEQTDKPGKKDRTTSKKKRTARKIAGFVVLGALVVVLGAGAWAAKTGIEMYRQTDLSKLAQEPPMPLRIYDVNKELI
TELTNSRMEYVPYSAFPKVLTNAIVAVEDSRFYEHDGIDYKGLSRALFTNVKNGEVVQGGSTITQQLAKVKLFSSEQTLE
RKINEAVAALKIENNYGKEQILELYLNYIYYGRGAWGIERASQIYFGKRTQELTLAEAALLAGLPKSPNAYSPLNNPDKA
KERRDLVLGMMASQGKITAAERDLAQNEPIKLTENTSLSASNKYPTYVDYVIQETLDKTKMSEQEILSAGLEIYTNLDPK
VQDAAETVYRDKKNFPGDKGGLQSSVVVLDAKTGAVKGLVGSRSETKEFRAFNYASQLQRQPGSTIKPVVVYTPALLAGF
KPGDLINDVETEFAGGYKPNNYGHRFHGWVTMEEALAQSYNVPAVAMMKEVGVEKALAFGRKAGLPLEDDDRVYGLALGG
MKYGTSPLVMAQAYTMYANGGSLAHAYAVTQVKDRNGNVILEQQTKTERVIDGDTAYTMSEMLQKVITEGSGRNARLNRD
VAGKTGTTQLPDVAEFRTSSGGVIDGSKDAWFVGYTPELVTAVWVGYPVTNREHYLTTGGDKLPAAIFQKIMSQALKGQP
VTAFKAPRGYRLAGGKIQRINGENGKAYAAVDSGERRSGSRTRTEDSTAGETPATPPPSPPSQTPADGTDTAPGGKETPA
AGTGSPPPKDGTGSVKPPVAGKEPPEGTEKPSGEATPPPSPGVKPPKGETAPTASTQPTSKPPAAKQEEGGAVGAGVSAS
RQSPVPSSSPAGEGAAVGN
>lcl|NZ_CP026520.1_prot_WP_084706597.1_4610 [locus_tag=PC41400_RS23255] [protein=aliphatic sulfonate ABC transporter substrate-binding protein] [protein_id=WP_084706597.1] [location=complement(5132989..5134062)] [gbkey=CDS]
MKAAHETVPAADERRRGYWRAIPFALLMTLLLSGCIGRGEQAITEGAAPAGQSGKDKVTVSIGIQQSLGPLLIAREKKWF
EEAFAKAGAEVKWQEFQSGPPHFESITAGRLDFGIVGNGPVIAAQAGKVEFKEIAVSGDGKKGNAILVPKGSPLQTVRDL
KGKKIAVAKGSSGWIFLYRAIVQAGLKPDDIKIVQLQPDEARPAFQTGAVDAWAVWEPFITSEVVTGGASVLADGKQLDL
LGPGFIVARSEFTREHPELTVLFLKTYEEARLWLTEHREEAVGILAKAKKLDPEIVRKVLSNLEALNAVITPEFVEAQQS
TADFMLLQDAIKQKIDVTKVVDNSFIEQALKQAAAGK
>lcl|NZ_CP026520.1_prot_WP_042230132.1_4611 [locus_tag=PC41400_RS23260] [protein=LLM class flavin-dependent oxidoreductase] [protein_id=WP_042230132.1] [location=complement(5134118..5135335)] [gbkey=CDS]
MGNAVSEVEFGWFIPTTGDGKYIGVEPERESSADYMIQVAQTAEEAGFTFALIPTGGSCLDAWIVGSIVASHTKTLKPLV
AMRPGLIAPVLAARMAATLDVLSGGRALINVVTGGSPSDLSAAGDPLAGDHDERYVRTREFLHIVRSVWTRSNDGGEKFL
AGGGYKYGDSEKVEFSGKYYRLEGGAALPLPVQKPYPPLYFGGSSEAGKRTAVETSDVYLMWAEPLAWIREQIREVEEIR
AEVKASTGKERELRYGLRAQVVVRETEEEAWKDAWEIISKVDDSAIEWSSHRFKETDATNQQRQNSLRESSRANGYVIGP
NLWAGISIVRGGGAMTFVGSAEQVADRLLEFTAIGISSFILSGYPNLEEARITGDLLLPVFKRKLAERGAAVDLAFSGHG
EKIEA
>lcl|NZ_CP026520.1_prot_WP_042230134.1_4612 [gene=mtaB] [locus_tag=PC41400_RS23265] [protein=tRNA (N(6)-L-threonylcarbamoyladenosine(37)-C(2))-methylthiotransferase MtaB] [protein_id=WP_042230134.1] [location=complement(5135639..5137000)] [gbkey=CDS]
MSTVAFHTLGCKVNFYDTEAIWQKFKGEGYEQVDFEQTADVYVINTCTVTNTGDKKSRQMIRRAVRRNPEAIVAVTGCYA
QTSPAEIMAIPGVDLVIGTQDRDKIISLVKQFQQERQPINAVRNIMKTRQFEELDVPDFADRTRAFLKIQEGCNNFCTFC
IIPWSRGLMRSREPESVVKQAHMLVDAGYKEIVLTGIHTGGYGEDMEDYSLARLLWDLDKVDGLNRIRISSIEASQITDE
VLEVLGASDKMCRHLHIPLQAGNDQVLARMRRKYTTAEYGRKIELIHKALPGAAITTDIIVGFPGETDEMFREGYAFMEQ
MKFAEMHVFPYSKRTGTPAARMLDQIDEEIKNARVHELIDLSERMQLEYAKQYVGKQLEVIPERAYKGAAEDSGLLMGYS
DNYIQMVFEAPEEMIGRVCTVEVQEAGVNESKGKLIRVAEEQSAAANRIKAIV
>lcl|NZ_CP026520.1_prot_WP_042230135.1_4613 [locus_tag=PC41400_RS23270] [protein=16S rRNA (uracil(1498)-N(3))-methyltransferase] [protein_id=WP_042230135.1] [location=complement(5137005..5137823)] [gbkey=CDS]
MQRYFIPAGDFGADSVAITGDDAHHLVRVMRAEEGDRCICSDGVSREAIVEITAVDKNRVEAKIVETLEMSAEPAVEVWI
AQSLPKGDKMETVIQKGTEIGAARFLPFVSERTVVQYDAKKEAKRAERWNKIAKEAAEQAHRNRIPVIEPVLSWKALLKL
VSEAADGAWICYEKEDGFGIGTAIRRYFGSRDSRTDEIGASRTAGADSAADGKPSKLLLLVGPEGGFTEQEIREAEAAGC
VSVSLGRRILRTETAAMVGLACILYESGEMGG
>lcl|NZ_CP026520.1_prot_WP_042230137.1_4614 [locus_tag=PC41400_RS23275] [protein=site-2 protease family protein] [protein_id=WP_042230137.1] [location=complement(5138003..5138707)] [gbkey=CDS]
MGDFNAILQLLPFTLLVLIIAFTVHEFAHAYSAYKFGDPTAYKEGRVTLNPRVHLDVLGTLLFVIAGFGWAKPVPVRRSN
FKKPRLMGVIVSFAGPFSNLVLGFLGLIVFYVLLNYTGIVGWSAGVQKAVLIFMILFVKMNVLLFILNLLPVPPLDGYRI
LEDVLPVAGREKLRQYEQWGMLVLLLVAFVPPLYRVTLGPIFSLQYDIIYAFNDLIGSLFGQRLPIDQFFRLIR
>lcl|NZ_CP026520.1_prot_WP_042230138.1_4615 [gene=prmA] [locus_tag=PC41400_RS23280] [protein=50S ribosomal protein L11 methyltransferase] [protein_id=WP_042230138.1] [location=complement(5138717..5139691)] [gbkey=CDS]
MRWHEITVNTTEQAVEMISNFLHEFGAGGVSIEESGTLNKKRDTSLGQWYELPFNDIPEGRAVIKGYFVEGTDLESIMTT
LKQSVEGLAEFGIDTGNPSYELREVDDEDWANAWKQYFKPVRITERLTIKPTWEEYTASEGEIILELDPGMAFGTGTHAT
TSLCLKTLEEVIKGGEDVIDVGTGSGILSIAAAKLGARRVLAVDLDPVAVSSASENSKLNGLEDQIHVKLSDLLGVFKES
EAGGGADLGVKLPVQVVVANILAEIILLFLDDVYKVLAPGGCYVVSGVILSKQKDVENGLLAAGFEVEDVRRDGEWVVLI
ARKR
>lcl|NZ_CP026520.1_prot_WP_042230140.1_4616 [locus_tag=PC41400_RS23285] [protein=hypothetical protein] [protein_id=WP_042230140.1] [location=complement(5140020..5140391)] [gbkey=CDS]
MSEKSILATFRSPEEAQGAEAKLKALRVIDMQIARFDRFQGEGVQRQMNPLTGDFASLGNLTQSAEWYSHDAGILAAADP
TASGMSDGGSGMPTGRDILLTVVVDESTHHQAMRVIRECGGEI
>lcl|NZ_CP026520.1_prot_WP_084706598.1_4617 [locus_tag=PC41400_RS23290] [protein=YfhD family protein] [protein_id=WP_084706598.1] [location=complement(5140388..5140543)] [gbkey=CDS]
MSENRNEQISQLIPIGKNEDVEFSSEEADAEDLEALQRANAADSRQERQGS
>lcl|NZ_CP026520.1_prot_WP_084706599.1_4618 [locus_tag=PC41400_RS23295] [protein=APC family permease] [protein_id=WP_084706599.1] [location=complement(5140679..5142022)] [gbkey=CDS]
MKPNMELQRTLKLWHIVVIGLGYMAPMAVFDTYGIVSEETGGHVPAAYALTLLAILFTASSYGKMVRVYPTAGSAYAYAR
NTIHPYAGFLVGWAALLDYLFLPMINALLTGIYLSAVFPEIPAWIWIVGFVLFITALNLLKVKITASVNTFLVLFQIVVV
ILFAALAVKGLMNGQGVGEVFTAQPFYAPDMSIPALLVGASILCFAFLGFDAVSTLTEETVDPEKNTPRAIILVAMIGGL
LFMTASYFTQSLFPDVSVFEDPEGASPEIALFIGGKVFQLVFLSAALSSTLASGLVSLTSVTRLLYAMGRDSYLPRKFFG
YVHPKTGTPVYNVLFIGCISLIALKLDLVTATSFINFGALIAFTFVNLSVIIHYIFKKKQRSFGDYMRYLVAPLIGAGFV
AYIWTHLDRHSMTLGLIWTAVGVVYLIYLVAVAKRDVKKIQMDDSGN
>lcl|NZ_CP026520.1_prot_WP_042230142.1_4619 [gene=dnaJ] [locus_tag=PC41400_RS23300] [protein=molecular chaperone DnaJ] [protein_id=WP_042230142.1] [location=complement(5142226..5143350)] [gbkey=CDS]
MSKRDYYEVLGVGKDASAEDVKKAYRKLARQYHPDVNKADDAETKFKEAKEAYDVLGDDQKRSTYDRFGHVDPNQGMGGG
GMGGADFGGFGDIFDMFFGGGGGGRRNPNAPQRGSDLQYNLQIEFKEAVFGKELDLQIPRTENCDKCSGSGAKPGTHPET
CTTCQGSGQQEVVQNTPFGRIVNRRPCSTCNGQGKIIRHKCGECGGSGKVKKQRKIHIKIPAGVDDGAQLRVSGEGESGT
RGGPAGDLYVVIHVKAHEYFERRDDDIYLDMPLTFVQAALGAEVEVPTLTEKVKLKIPAGTDTGTYFRLRGKGVPKLRGY
GQGDQHVRVVIVTPKNMTEEQKDLLREFAKVSGEDTKEHEESFFDRMRKAFRGD
>lcl|NZ_CP026520.1_prot_WP_042230144.1_4620 [gene=dnaK] [locus_tag=PC41400_RS23305] [protein=molecular chaperone DnaK] [protein_id=WP_042230144.1] [location=complement(5143480..5145324)] [gbkey=CDS]
MSKVIGIDLGTTNSCVAVMEGGEAVVIPNAEGNRTTPSVVGFKKDGERVVGETAKRQSITNPDRTIISIKRHIGTNHKES
IDGKDYTPQEISAIILQKLKADAEAYLGSPVTQAVITVPAYFNDSQRQATKDAGKIAGLEVLRIVNEPTAAALAYGLEKS
EDQTILVYDLGGGTFDVSILELGDGFFEVKATSGDNKLGGDDFDQVIIDHLTAEFKKEHGIDLSKDKAAVQRLKDAAEKA
KKELSSVLTTTVSLPFITVVDGVPQHLELNLTRAKFEELSASLVERTLGPTRQALSDAGLSPSQIDKVVLVGGSTRIPAV
VEEIKKLIGKDPHKGVNPDEVVALGAAVQAGVLTGDVKDVVLLDVTPLSLGIETAGGVFTKMIDRNTTIPTSKSQVYTTY
ADNQTSVEIHVLQGERSMANDNKTLGRFMLGDIPPAPRGIPQVEVTFDIDANGIVNVSALDKGTGKSQKITITSSSGLSD
EEIDKMMKDAEAHAEEDRNRKELVEVRNQADQLVYSTDKTIKELGDKVDQAEIDKANAAKEKVKQALEGNDLDAIKAATE
ELTEVVQQLSVKLYEQAAQQQGAGDAAGEASDAGKKDNVVDADYEVVDEDKDKK
>lcl|NZ_CP026520.1_prot_WP_042230145.1_4621 [gene=grpE] [locus_tag=PC41400_RS23310] [protein=nucleotide exchange factor GrpE] [protein_id=WP_042230145.1] [location=complement(5145392..5145967)] [gbkey=CDS]
MSTEETKNINPDTNAAEAEKATYDAEQAEAAEAAAPDAEGATNAELDELRKAAEESQQRLLRAQADFDNFRRRTRQEKEE
FAKYASLKLIEQMLPVIDNFDRALVSSRETQDFEALTKGIEMVYRQLDQVMTQEGLTPIEAVGQPFNPEFHQAIMQVESD
EHEEGIVVEEVQKGYMLKDKVIRPSMVKVSM
>lcl|NZ_CP026520.1_prot_WP_042230147.1_4622 [gene=hrcA] [locus_tag=PC41400_RS23315] [protein=heat-inducible transcriptional repressor HrcA] [protein_id=WP_042230147.1] [location=complement(5146085..5147110)] [gbkey=CDS]
MLTERQRMILNAIVDDYIRSAEPIGSRSISKRGDVGFSPATIRNEMSDLEELGFLEQPHTSAGRIPSHKGYRYYVDHLVR
HGTLDAHELETVKLFFAEKVQEMEEVIQHTANMLSGLTNYTSIVLGPEVFTNTLKHLQIIPLNDKTAVAIIVTNTGHVEN
KTVAIPPGLPMNEIEKVVNLLNARLQGVPLLQFKSKLYTEVAGELNKYAVGYHELLTMVESVLENDEKDRIFLGGTTNML
TQPEFKDVDKVKNILDLLEQTPVLFKLVTSSPTGLQIKIGSENSLAAMNDCSLITATYSYDDQPLGTIGILGPTRMEYAK
VIRLLEHLSANMEVLLSRWYK
>lcl|NZ_CP026520.1_prot_WP_042230149.1_4623 [locus_tag=PC41400_RS23320] [protein=N-acetyltransferase] [protein_id=WP_042230149.1] [location=complement(5147248..5147709)] [gbkey=CDS]
MTVACRRATEDDIEELVHIIQGYARQGIMLPRDREMVRQHLDTFVVAEDEGALIGCGSLTRLGENLVEIRSLGVSEDYKG
QGVGTKLVDSLLEEAQRLSVTKVMALTYEVKFFQKNGFRIVPKHIFPEKVWTDCIHCKKQYCCDEIAVLRELD
>lcl|NZ_CP026520.1_prot_WP_042230150.1_4624 [gene=hemW] [locus_tag=PC41400_RS23325] [protein=radical SAM family heme chaperone HemW] [protein_id=WP_042230150.1] [location=complement(5147910..5149061)] [gbkey=CDS]
MNTRYTAPEAVYIHIPFCTNKCFYCDFNSYVLKGQPVMDYLDALEREMERTVQAQPPGKIATIFVGGGTPTVLLPDQMAH
FLKLVRTYFPDQHENLEFSMEANPGTTDLEKLTAMKEGGVNRISFGVQSFDNGLLETIGRIHNTDDVYRSLENAAKAGFD
NMSIDLMFGLPNQTPEMMEKTLNAALALDLKHYSIYSLKVEENTLFHTMYHKNQLPLPSEDDELDMYLLIMRRLKEAGYG
QYEISNFARPGYESRHNTTYWRNQPYYGLGAGAHGYAEGHRHVNVKGVQPYIDRTKEGLPHLERFPVTEQEAMEDFMMVG
LRLLEGISGDDFTAQFGIPVEEKFGKTLEDLISRGLLERSGNKIRLSEQGILFGNDVFASFLE
>lcl|NZ_CP026520.1_prot_WP_042230151.1_4625 [gene=lepA] [locus_tag=PC41400_RS23330] [protein=translation elongation factor 4] [protein_id=WP_042230151.1] [location=complement(5149191..5151008)] [gbkey=CDS]
MTDIRQRQKQIRNFSIIAHIDHGKSTLADRILEFTGALSSREMQEQVLDTMDLERERGITIKLQAVRLNYKADDGQEYIL
NLIDTPGHVDFTYEVSRSMAACEGALLVVDAAQGIEAQTLANVYLALDNNLEILPVINKIDLPSADPERVKQEVEDVIGL
DASEAVLASAKAGIGIKEILEQVCQKVPAPQGDPDEPLKALIFDSHYDPYKGVIVYVRVVDGAIKAGTKIKFMATGKVFD
VIEVGAFKPRMSIVDELNVGDVGFVVAGIKNVGDTRVGDTVTNALKPAAEPLPGYRKINPMVFCGLYPIETSDYNDLREA
LEKLELNDASLTYEPETSSALGFGFRCGFLGLLHMEIIQERIEREFNIPLITTAPSVIYRITMTNGEIIEIDNPSNYPDP
QRIDFVEEPYVKASVIVPNDFVGAIMELCQGKRGEFINMEYLDTNRVTLSYDIPLAEIVYDFFDLLKSRTKGYASFDYEI
SGYKKSNLVKMDIMLNGEQVDALSFIVHRDGAYNRGRIICEKLKELIPRQMFEVPIQAAIGQKIISRETVKAMRKNVLAK
CYGGDISRKRKLLEKQKEGKKRMKQVGSVEVPQEAFMAVLRIDDN
>lcl|NZ_CP026520.1_prot_WP_042230152.1_4626 [locus_tag=PC41400_RS23335] [protein=hypothetical protein] [protein_id=WP_042230152.1] [location=complement(5151100..5151615)] [gbkey=CDS]
MPRWYWKLFVYGIILCFCIMFGVDLASRGMNRIEGPGGGSGGAARTGGSGGPATSGSLPEETFIRTGGGRSGGQQPGVTS
QAQGTGASAPSPAAGPDAAAGNAGSARTAESANRPAETPAPPKPTPRPLGTGDPLEAGSMNHLGNKLGDLLQITAQSGMD
GVVSMFEWVVK
>lcl|NZ_CP026520.1_prot_WP_042230153.1_4627 [locus_tag=PC41400_RS23340] [protein=stage II sporulation protein P] [protein_id=WP_042230153.1] [location=complement(5151703..5152914)] [gbkey=CDS]
MKRNAVTFNLSKIRQTFQSAGVIGKTTISLSLFTLCLFILLGVGSMLQSKLSTSPVSSMKGLAASVSNQFFLDMLGLEVP
HLRNDSRKFTFSQQNMMSFAVRFLTDVNPGDPKSLIAQEVPGLKADKSTVLRSGKDGDASGPVEYTPAEGAVNGTTGGTA
PEASSAPSPAPSQAPADQKPKGPGRNIAFIYQSHSNESFLPELKGVTDPDKAYDAKTNVTMVGKKLAEALEKEGVGAVHS
DAIYPNMIKDFKYAYSYKYSLQTLEAAANAHPDLKFFFDIHRDSQERKRTTVTLNGKDYAQVYFILGGDNPKWKENEAFA
GKIHEVLEAKAPGVSKGIHAKTGHEVNGVYNQNFSPNSVLIEVGGPYNSLEECYRTVELLAKAVSEVILDAEKVTAPAQG
KKS
>lcl|NZ_CP026520.1_prot_WP_009675370.1_4628 [gene=gpr] [locus_tag=PC41400_RS23345] [protein=GPR endopeptidase] [protein_id=WP_009675370.1] [location=complement(5153111..5154112)] [gbkey=CDS]
MADLSAYTVRTDLALEARELASEARKSPEIPGVVSDIWEDGGMRITQIQVLNDEGAQAIGKMPGQYMTIEVPDLRKKDTQ
LQDRVATKFAQVFEQYLQKAGIGKDSKVLIVGLGNWNVTPDALGPLVVENILVTRHYFELMPGQVSPGYRPVSAVAPGVL
GTTGIESSEIVEGIVEKSRPDLVIAVDALASRALERVNTTIQIADTGINPGSGIGNKRRGLTLETLGVPVIAIGVPTVVY
ASTIVNSCIDMMQHHFRTQTSNTHEILGMLDDLPEQERLSLVREVLNPLGHDLLVTPKEIDEFIEDIANIIASGLNAALH
EAVDSANVAAYTH
>lcl|NZ_CP026520.1_prot_WP_042230156.1_4629 [gene=rpsT] [locus_tag=PC41400_RS23350] [protein=30S ribosomal protein S20] [protein_id=WP_042230156.1] [location=5154317..5154589] [gbkey=CDS]
MPNIKSAIKRVKVSEKRRLRNASAKSALRTAVKAFETTVASNNVDSAKEALINATKKLDKAASKGLIHKNAASRKKSRLA
KKLNALSAQG
>lcl|NZ_CP026520.1_prot_WP_042230158.1_4630 [gene=holA] [locus_tag=PC41400_RS23355] [protein=DNA polymerase III subunit delta] [protein_id=WP_042230158.1] [location=complement(5154891..5155910)] [gbkey=CDS]
MDYKQAMKSIDEQKFAPVYVAYGSESYLIEQFIRRLTDKMIEPEHLDFAVSKYDLNEVPVQNVIADAETLPFMVPRKLII
AGNAQFLTGSRDSGKVDHQLDRLTEYLGAPVDFTVLVFTVSVDKLDERKKLVKQLKEKKALVPCLPLTAEELQAWVRRQA
KRESLELGDGVVDQLVLYTGGHLQSLAAEIEKISLNVGRGGTATPELIDSLVVRSTEQNVFILIEQVVQLRLDRAFALLE
ELLKQKEEPIKIVMLMARQFRIMLQVKELSKQSYSHQQIAGQLGLHPYAVKVAEGQARAYDVEKLSRIVSQLADLDFKMK
SGGIDKVLGLELFLLGLAA
>lcl|NZ_CP026520.1_prot_WP_042230160.1_4631 [locus_tag=PC41400_RS23360] [protein=zf-HC2 domain-containing protein] [protein_id=WP_042230160.1] [location=complement(5156234..5157430)] [gbkey=CDS]
MNCKEVKELMQRDLDGDLNDLERQRLNQHVQECSECGTMLQRLRLLSKELENLPKIMPPFSIVDSILPQLERLDEEKSNR
REEAAGLSPAVGRSPRSPEQLPWTRRIAAHVSWKTVSGVVAAGFVLGFFAFNMDRPAQSGPAGMFSALTGTNTENADTGA
KAVQPAIIPDENASGKQSGGTPTAPPATHTPGDSKSAGKSQTDSNGEANRTDRTASTGDTDKGSGAEQQKSGSDSPKEEG
SRKEQPGTTSTPQPNKQSDASANEGKAPVGTDNPTVDRGVSPGSGDTSGGTSGSGDGQTGGKIATTDPDKKDVHTFTAAG
TGGTSPDGKLSAKIEEHKLTIVKTDGTSVYVSAKQWKETDTLRLVGWNGNDEFVYEVSSGNTKQEYVVSLKTKQERAK
>lcl|NZ_CP026520.1_prot_WP_009675391.1_4632 [locus_tag=PC41400_RS23365] [protein=RNA polymerase sigma factor] [protein_id=WP_009675391.1] [location=complement(5157427..5157975)] [gbkey=CDS]
MVAPEMVKAAQSGDREALVTLLRQIETHVYRTAYYLLGNEQDALDASQEALIRIYTKIQSYEEKALFKTWVQRIVTNICI
DKFRKTKPTVSIEEHELTFTTPKDVEDEVMIGYLAQDIQEAIEKLPEHHRSVVVLRYLQDFSYHEIADCLGLPLNTVKSY
LFRARQQLQGLLHEYQKGGVRG
>lcl|NZ_CP026520.1_prot_WP_042230163.1_4633 [locus_tag=PC41400_RS23370] [protein=ComEC/Rec2 family competence protein] [protein_id=WP_042230163.1] [location=complement(5158538..5161366)] [gbkey=CDS]
MRLNRNDIVGLAVLWVIGYAAAIYASKPWMTAWSAAAFAAAAAGVMALRLPKRRWIAAALVAAAACCYYVDTDRRNSTAL
PAAAAAAAHGSQVSVAGTLASPVAVDGDRASFELRVSQLGESGSEAGAAEGAPPEERVQVSLRLLSPAEQTAAASWRRGD
RLAMQGELQEPAAARNFGGFDYRSYLRQQGKHWLLVVKGVQQVQHGPPAPISAALTSPRLAAERLLRTTDALRGYLGSRI
DEIFTPEQAGFMKGMLIGLTDDMDPGRYSQFSNLGLTHILAISGLNVAVFLGVLIWMLKRLGLPKETYLLVAVFLLPFYI
LLTGASPSIVRAGLMTMIGLLAVRKNRIKDILQIACLVGLGLLIWNPYYLLDVSFQLSFLVTLGLIVGVPRVTALLPLKN
GVLRNTAAITLVSQLVSFPVSIYYFNQFSLLSWLANAALVPVFSLLIFPAGLIATCLGLVYVPLGRLGAWVTEMGTGFVF
RIVEWLDQWEGFRTIWPSPSVFWIVVYYLLLAGICGGLERLVRSRRVQNEDGTPQLDAPAPGRSMFTRYGTSSAAVIALG
LGWLGLLYYGYTPDRWSSEAYVDMIDVGQGDSILIRTPAHKYILVDGGGTVSFRKPGEEWKLRRDPYEVGRKTLVPLLKK
RGVHKLDAVILTHEDADHSGGLQAVLEDIPVEAFIFNGTYKSGPQVEKLFRTVIDKKIPLISAAYGKTLRMDNRTRLDFL
APLPENGQLMAGRDSSRDASGSGQPAFPDQRTAGKTGAEFAPSAAQDNYGSTDNPIVLENEQNGVSVVFLMEMEGTRWLF
TGDMDVKAERRVMELLQSRNSVPGTHGTKLPQSDTGAKEPDGTEDGTPQTPGIDVLKVAHHGSKTSTGSEWLGYWKPRTA
LISAGVKNMYKHPSPDVVARLERNDIRVLRTDLLGEIQLLVKNGEIYVRSVLSGKPAASGDS
>lcl|NZ_CP026520.1_prot_WP_042230165.1_4634 [locus_tag=PC41400_RS23375] [protein=dCMP deaminase family protein] [protein_id=WP_042230165.1] [location=complement(5161949..5162461)] [gbkey=CDS]
MTGRKDWDTYFMDIAYMASTRSQCSRRHVGAVLVQGKKLLGTAYNGAPMGVPDCSEEGCMLVEEYELHVADGKEEMIRKQ
RCIRTIHAEQNLLLFTDREDREGSSVYVTDQPCWTCSNMLANSGVREIVFHRPYPKDSEKVSALMAARGIAFRQIQGYEP
PEGTDHRIVN
>lcl|NZ_CP026520.1_prot_WP_042230167.1_4635 [locus_tag=PC41400_RS23380] [protein=homocysteine synthase] [protein_id=WP_042230167.1] [location=complement(5162470..5163762)] [gbkey=CDS]
MSEQRPYSLETLAVHAGQQIDPTTLSRAVPIYQTTSYGFRDADHAADLFALKEFGNIYTRIMNPTSDVFEQRITALEGGA
GALATASGQAAITYSILNIAGAGDEIVSASSLYGGTYNLFSTTLAKLGIQVKFVDPADPENFRKAITDKTKALYAETVGN
PRGDLLDIEAVAAIAHENGIPLIVDNTFPTPYLLRPIEHGADIVVHSATKFIGGHGTSIGGVIVDGGKFDWKASGKFPGL
TEPDPSYHGVVYADAVGPIAYIIKARVQLLRDMGASISPFNSFMLLQGLETLHLRMERHSSNALAVARFLEAHEAVDWVS
YAGLESHSSYKLAQKYLPKGQGAILTFGIKGGLEAGRKVIEQVKLFSHLANVGDSKSLIIHPASTTHQQLSEEEQLSAGV
TPGMIRLSIGTEAIDDILHDLEQAILASQQ
>lcl|NZ_CP026520.1_prot_WP_241688723.1_4636 [locus_tag=PC41400_RS23385] [protein=helix-hairpin-helix domain-containing protein] [protein_id=WP_241688723.1] [location=complement(5163927..5164631)] [gbkey=CDS]
MNWNFQSWKSGGVLAGIAAVIAAGLGIWLLVGPRGTVPQGFTALNASMEQTIEATKPARPSEKPLSSPVKAEAKEEAGAG
SKQAGAEESAAGAAGGGPEQAPASARKEAAADDKPAQGDKRQPVGGGNGPAPGLQESGKLGSGAAVPVPETSTGGGPKGS
TVDKKIPAGPKIDINTANLQQLMELPGIGESKGKAILAYREQHGKFKRVEQLLDIKGIGEKMLAKMKPYLQINP
>lcl|NZ_CP026520.1_prot_WP_042230169.1_4637 [gene=comER] [locus_tag=PC41400_RS23390] [protein=late competence protein ComER] [protein_id=WP_042230169.1] [location=5164875..5165705] [gbkey=CDS]
MKAGFIGTGSMGSILIEALIHSGALNPEQICAANRTPGKVQRLAETYAGLTAASSNAEAAAGRDVVFLCVKPLEYMRVID
EIRPVIEPAQIVVSITSPVLLKHLEDQLPCKVTKIIPSITNYVLSGSTLCIHGERMHPEDKEWIENLLTHISTPVRVSEA
YTRISSDLSSCAPAFLSFFLQKYIEAAVKETGISSEEATILASEMTLGTGKLLTTGGFDPVSLQKRVRVPGGITAEGLQL
MEQDLDGMFNRLIGITHAKYEEDLEKVEAVFTTKKQ
>lcl|NZ_CP026520.1_prot_WP_042230171.1_4638 [gene=leuS] [locus_tag=PC41400_RS23395] [protein=leucine--tRNA ligase] [protein_id=WP_042230171.1] [location=complement(5166013..5168460)] [gbkey=CDS]
MAHEKEHQGYTPQVIEPKWQKYWDDNQTFRTSEEAGKPKFYALDMFPYPSGAGLHVGHPEGYTATDIVSRFKRMRGFNVL
HPMGWDAFGLPAEQYALDTGNDPREFTKQNIENFRRQIKSLGFSYDWEREISTTDPEYYKWTQWIFIQLYKKGLAYVDEV
PVNWCPALGTVLANEEVIDGKSERGNHPVIRKPMRQWVLRITEYAERLLEDLEELDWSESIKDMQRNWIGKSKGAEVTFA
IDGHEGDSLTVFTTRPDTLFGATYCVLAPEHELVAKITASGQEADVKAYQEKAARKSDLERTDLAKEKSGVFTGAYAVNP
VNGAKLPIWIADYVLAGYGTGAIMAVPGHDERDWEFAKQYDLPIIEVVQGGSVQEAAYDGEGPHVNSGFIDGMGKEEAIA
SMIAWLEENGKGKGKVTYRLRDWLFSRQRYWGEPIPILHLEDGTMKTVPEDQLPLLLPQVEHIKPSGTGESPLANVEDWV
NTVDPETGMKARRETNTMPQWAGSCWYYLRFIDPRNNDAICSPEKQKEWLPVDLYIGGAEHAVLHLLYARFWHKVLYDLG
VVQTKEPFHKLVNQGMILGENMEKMSKSRGNVVNPDDIVNEFGADTLRMYEMFMGPLEATKPWNTNGVEGSYRFLNRVWR
LFVDENGALSSKISEDEKAGGDAFKRTWHRTVKKITEDFEALRFNTAISQLMIFVNEAYKTEALPKTAVKHFVQMLSPLA
PHITEELWQKLGGTGSITYEAWPVYEEAWTVDDEVEIVVQVNGKIADRVKVAKDLDEAGMQELALGLDKVKELSAGKTVR
KIIAVKGKLVNIVVG
>lcl|NZ_CP026520.1_prot_WP_255246841.1_4639 [locus_tag=PC41400_RS30635] [protein=hypothetical protein] [protein_id=WP_255246841.1] [location=complement(5168624..5168755)] [gbkey=CDS]
MMKASSPPSRKANRSTAVSLQLCLSSGRKPDGSEDVLPSNRTV
>lcl|NZ_CP026520.1_prot_WP_042230174.1_4640 [locus_tag=PC41400_RS23400] [protein=class I SAM-dependent methyltransferase] [protein_id=WP_042230174.1] [location=complement(5168893..5169681)] [gbkey=CDS]
MSYRDFAYTYDRLMEDMPYEAWISYARQAWESYGIRPETVVDLGCGTGNIAVPLAKLGCKVVGIDLSDDMLAVARHKAEE
EAAIGAKPDILWLQQDMRDWELPEPADTVISFCDCFNYLLEEEEIIEALEQAYRGLRSGGVLLFDVHSPAQFEEYASLQP
FVFDEEDVAYLWTCELDEERTEIEHALTIFVREDASSARGGEPLFRRIEETHRQRAYDPRWLKERLEAAGFEDIRISADF
DWEAEFSPDSRRLFYAARKPLS
>lcl|NZ_CP026520.1_prot_WP_042230176.1_4641 [locus_tag=PC41400_RS23405] [protein=S1-like domain-containing RNA-binding protein] [protein_id=WP_042230176.1] [location=complement(5169683..5170567)] [gbkey=CDS]
MSLEAGTLVTLDVTREVSPYGYFVTDGEREVLLPYAETNRKLKAGQQAEVFIYYDSEDRMAATMNRPMLLLGELALLEVV
DLHPRFGAFLEMGIGRNLLLPFKEMPELHELRPQVGDKVFVTMDHDRQGRLVARLAGEDVLSEKCFPAPQAWRNQWLDAR
VYKPLQMGTFVVCDGGVVGFGVIGLIPSTERSRLLRVGEEVSVRVTYIREDGRVNLSMRQPKEKGRDEDAERVLAVLRER
PGGAMPYSDQTEADLIKSKFGISKSAFKRALGKLMKDGLVYQEENWTYLKQEEN
>lcl|NZ_CP026520.1_prot_WP_042230178.1_4642 [gene=rsfS] [locus_tag=PC41400_RS23410] [protein=ribosome silencing factor] [protein_id=WP_042230178.1] [location=complement(5170564..5170911)] [gbkey=CDS]
MTVNPEQLMNLVVDAAEDKKAINVVALDLQEISLVADYFVICNGNSETQVQAIATEIRKKADEHGVTVRGLEGFDTARWV
LVDLGDVVVHVFHRDDREYYNIERLWSDARVVERV
>lcl|NZ_CP026520.1_prot_WP_042230180.1_4643 [gene=yqeK] [locus_tag=PC41400_RS23415] [protein=bis(5'-nucleosyl)-tetraphosphatase (symmetrical) YqeK] [protein_id=WP_042230180.1] [location=complement(5171045..5171611)] [gbkey=CDS]
MNREEMMKTVKEEMPEKRWRHTLGVMETAVILARRFGADEEKADRAAILHDVCKYWPVEKQADAIRESGFQTDVLSFDKE
LWHAHAGAWIASDRFDVKDEDVLNAIRYHTSGRENMSTLEKVVCLADYMEPGRDFPGVDKIRELAETSLERALLAGFDST
LSFLLQKGRRVYPLTVLTRNRLILELDV
>lcl|NZ_CP026520.1_prot_WP_042230181.1_4644 [gene=nadD] [locus_tag=PC41400_RS23420] [protein=nicotinate-nucleotide adenylyltransferase] [protein_id=WP_042230181.1] [location=complement(5171601..5172188)] [gbkey=CDS]
MRIGIMGGTFDPVHTGHLVAGECARAACGLDEVWFMPANVPPHKPNAPKASAEQRWEMVCRAVSDNPAFRPFDHELRKGG
VSYSVDTIRELKARYPEHEFHYIIGADMVEYLPKWREIGEIMKMIGFIGLQRPGFELQPEALPAFIRPGVRLAPMPLLDI
SSTDIRTRRREGRSVRYLVTAPVHDYIEGMGLYES
>lcl|NZ_CP026520.1_prot_WP_042230182.1_4645 [gene=yhbY] [locus_tag=PC41400_RS23425] [protein=ribosome assembly RNA-binding protein YhbY] [protein_id=WP_042230182.1] [location=complement(5172309..5172605)] [gbkey=CDS]
MLTGKQKRHLRSLAHHLNPVFQVGKGGVNEHLIQHIVEAIEVRELMKISVLNNYEADCKEVARELAEGAGAEVVQVIGRT
IVLYKESRDNKQIELPKS
>lcl|NZ_CP026520.1_prot_WP_042230183.1_4646 [gene=aroE] [locus_tag=PC41400_RS23430] [protein=shikimate dehydrogenase] [protein_id=WP_042230183.1] [location=complement(5172620..5173489)] [gbkey=CDS]
MGNKPSLDSGTILYGVFGDPVRHSRSPIMMNRAFGETGINGAYAAFHVLPEQLEEAVAGIRALGFGGVNVTIPHKVNVMK
YLDEINADAEALGAVNTVVNRGGKLIGYNTDGIGYVRSLKEETGMDLRGKKVLLIGAGGAARGISYALAKEGPSFIAIAN
RTAEKAQELASTVRSETSAVGMGLHELGNIISDADLIINSTSSGMYPNIEEMPLDLDGHSLREGAVVSDLVYNPLETRFL
KEAKAKGAVTHGGLGMFIYQGAYAFEYWTGVPAPAAAMRQAVEQSFIGN
>lcl|NZ_CP026520.1_prot_WP_042230185.1_4647 [gene=yqeH] [locus_tag=PC41400_RS23435] [protein=ribosome biogenesis GTPase YqeH] [protein_id=WP_042230185.1] [location=complement(5173574..5174689)] [gbkey=CDS]
MTQEEVIHCGGCGVKLQLEDPHALGYIPAQALEKEPVICQRCFRIKNYNEASSITLNQDDFLKLLGHVGHKEALVINIVD
IFDFEGSLISGLARFVGGNPIILVVNKIDLLPKVTNWNRIVNWVRKQAKEQGLKVVEVVLCSAKKNMGFDRVLQAVDEHR
SGKDVYVVGATNVGKSTLINRLISDYSDLDSELTTSQYPGTTLDMVEIPLDDGHFIIDTPGIVYKHRLTELVSKKDLGKL
MPGKALKPMVYQLNEGQTLFFGSLARFDFVRGERQSFTIYSSGAVPIHRTKLEKADELYANHKGTLLTPPVVKDLEELPP
LVKQTIHISKGKKLDVLISGLGWIKINSEAGAELAVHVPKGVKIVQRDSLI
>lcl|NZ_CP026520.1_prot_WP_036739818.1_4648 [locus_tag=PC41400_RS23440] [protein=YqeG family HAD IIIA-type phosphatase] [protein_id=WP_036739818.1] [location=complement(5174776..5175276)] [gbkey=CDS]
MFKKLVPKQFVQTIYDINLEELWQSGIRGIITDLDNTLVGAKVADATPELVNWLGHVKQLGFKVVIVSNNQETRVSTFAL
PLSIPFIHAARKPTNTAFRKALALMDTTAEQTVVIGDQMLTDVLGGNRLGLYTILVAPISIHDEGFFTRINRRIEKLANN
YISKNK
>lcl|NZ_CP026520.1_prot_WP_042230188.1_4649 [locus_tag=PC41400_RS23445] [protein=transglutaminase domain-containing protein] [protein_id=WP_042230188.1] [location=complement(5175543..5177810)] [gbkey=CDS]
MRSIWNVIRFMAGEDGTKKWMALFVLLFLFQFTGWISREDGLWLPETLKAVNWTLITVCATFFIPRIHHLVRELLQIVLV
LVVHAIVLQYRFVPMSQMDGPMEWFQANVSQLFPYLWFGLAAWWVFLMAMWASRSKLWISFIVILAILFFGVRDSFSTLQ
LWPQVAAVIFSGLCMLVVRHLSQLKNKAPESWKRLSDNPGPLIVPVVLIIALTLTLGALVPAVDPVLTDPYTAWRQYNGE
AVTFLPSSDDSGRTMVQDPNDPLLKSSGYSRNDAQLGGQFQYDYTPVFTVDTTSRTYYRGETRALYTGSGWEASPADRRA
RQTSVDSSGKLPADSGIDMTKLKTEEVKQTVKMRSNEVYPVLFGAYPMSKVDIQEEGPDGTRSDPAQLGSLRWSARQAEL
RFNAKSNYPKQYELVSNIPLLDLSGLKKVDMSRLNRSEWNDYLQLPDRLPERVTELAKEVTKEAPSAYEKVHELELYLSS
KYPYTNTPDVSKGKSKDFVDRFLFEIQEGYCDYYSSAMVVMTRSLGIPARWVKGYASGSSDNSRPAGIGNLRGGMSDPDA
EGTYTVRNSDAHSWVEVYFEGYGWLPFEPTAGFSLPQSYAPETSAATEMPTPAPAETKTAQAKSGAAPWVRYVLYTLSGL
VLLTAAAVTAIRLSSGKKLLPLLGRDPKAANYAQGVIQEVNRLLRYGRRKGYERFEHETFREAASRWMTQSYWLAGELSP
VLAMFEKAKYSRSSVTEQDLEAVQGHVKRLREQMK
>lcl|NZ_CP026520.1_prot_WP_042230189.1_4650 [locus_tag=PC41400_RS23450] [protein=DUF58 domain-containing protein] [protein_id=WP_042230189.1] [location=complement(5177803..5179050)] [gbkey=CDS]
MRLFKPTAAQRGRQATLILSFLFFASLMFLLFQGGKLASTLFAVVCLLSAYLVLGKWSGIRRTQAVREISAADGTGKVQA
GQSLQIKLKIQVPGVWPIPYLLIEDELVRSSGERTVMKGSVVLNWMRRGEFVYKTPPLRRGHYSLERAEFATEDVFGLYE
HKGSVQLPYGFVVMPQMIEIRHWAQFSQSPRGARHHSVTARMQRETTQINGVREYIHGDRLSRVHWNATARTGTWKSKEF
ERETLPKMVIVLDRSKQNYGSPAEFELAVSVAASLLHFGTQRNIAVGLVSSGRESLMAVPKTGSRHEDVLMDHLVEVEPD
GFRPLQQAVNNADGMFSPGALVVYITPAKNSDLLSLLHWTDKQRMLACHLCVCGAKGGPADVWVRRLHSLGFMGYGVHTL
AELPAVLGGGTKIYA
>lcl|NZ_CP026520.1_prot_WP_009675380.1_4651 [locus_tag=PC41400_RS23455] [protein=MoxR family ATPase] [protein_id=WP_009675380.1] [location=complement(5179047..5180012)] [gbkey=CDS]
MAAVSQPHMSEISRIRSNLDSCILGKSEEIMLLLTALLAGGHVLLEDVPGTGKTVLVKSLARSIQGQFRRIQCNPDLLPT
DITGVSIYHPKNEAFFFRPGPVMANILLVDEINRATTKTQSALLEAMEEHHVTVDGEMYPLPRPFMLLATQNPIDFEGTY
LLPEAQLDRFMIKFSLGYPDSATELQMIASQSKEHPMEQLHEVVGAEQILSMQEEVRSVYLDEGIAKYLVDITRSTREHP
AVELGASPRATLALVQASKAYAYLNGRDYVLPDDVKYLAPYVLGHRIILSSEARMEGETSRSVFASILAHTKVPVRLEKA
K
>lcl|NZ_CP026520.1_prot_WP_042230191.1_4652 [gene=spoVAE] [locus_tag=PC41400_RS23460] [protein=stage V sporulation protein AE] [protein_id=WP_042230191.1] [location=complement(5180331..5180681)] [gbkey=CDS]
MQFLWAFLVGGAICVIGQLLLDLGKLTPAHTMTTLVVAGAILDGFGLYEPLVRFAGAGASVPITSFGNALVHGAIKEIQT
SGWIGIVTGIFKVTSAGISAAIIFSFLAALFVRPKG
>lcl|NZ_CP026520.1_prot_WP_174446818.1_4653 [gene=spoVAD] [locus_tag=PC41400_RS23465] [protein=stage V sporulation protein AD] [protein_id=WP_174446818.1] [location=complement(5180684..5181700)] [gbkey=CDS]
MIRKGQSWEFRKPPVLISTATVVGPDEGEGPLALDFDLIHQDNTLMQKSWEKAEKALMEDAANLAIKKAGLQKDQVQFYV
GGDLMNQIISSSFAARTLAIPYLGVFGACSTSMESLAVSSLIVASGGGDYVLSGTCSHNCTVEKQFRYPTEYGSQKPPTA
QYTVTGAGASVVAEKGTGPVVTSATIGKVIDMGITDPFNMGAAMAPAAADTITAHFRDMKRSPDYYDLIVTGDLATVGFQ
IVNELLKEQRIAMNSTTFSDCGLMIYDIEKQGVQAGGSGCGCSAVVTYGHLLKRMEQGDIRRLLVVATGALLSPLSYQQG
ESIPCIAHAVAIERQEGL
>lcl|NZ_CP026520.1_prot_WP_042230195.1_4654 [gene=spoVAC] [locus_tag=PC41400_RS23470] [protein=stage V sporulation protein AC] [protein_id=WP_042230195.1] [location=complement(5181700..5182176)] [gbkey=CDS]
MANQKKKNATLTQQQYQAIAKKNEPKRPVVANCIRAFVVGGFICIIGQALMMLFEKVFGFSKEAASNPTVSVLILISVIL
TCLGVYDKIAQWAGAGTAVPVTGFANSMCSAAIEHRSEGLILGTASNMFKLAGSVIVFGVVAAFFVGIFTLIFGGGAH
>lcl|NZ_CP026520.1_prot_WP_042230197.1_4655 [locus_tag=PC41400_RS23475] [protein=DUF421 domain-containing protein] [protein_id=WP_042230197.1] [location=complement(5182195..5183070)] [gbkey=CDS]
MEGIGVIVLRSLGSIAMLFAITRILGKKQISQLSFFEYVTGITLGELAGFLSTDIEAHYLHGVVALGIWFVVPFLLELTT
LKSKWLRHFFEGKGTVVIRQGRVLEHHLKKERYTADELMEQLRTKSIFQVADVEFAMLEASGELSVLLKKEKQPLTPGDL
GLKMGPDKLPHTVIVDGVVDESELAASSLTREWLNKQLAKINATADNVFLAQIGDQNKLYVDLYDDSKGISLEKPQVELI
STLRQCTADLDSFANSATDLNARIKYEQTASALRDVLKQLSGGGKSTNGTE
>lcl|NZ_CP026520.1_prot_WP_042230198.1_4656 [locus_tag=PC41400_RS23480] [protein=carboxypeptidase regulatory-like domain-containing protein] [protein_id=WP_042230198.1] [location=5183490..5189201] [gbkey=CDS]
MSFPSDSQFFPVTLNGVPVGDPVADVSPASTDIVGNTQFPAAYVSYDTTYVYFRLRLNGDPRAKTGFDNFAWGVLFNTDS
NPATYEWVLAVNGQNASIDLIENKIKVPNTFNDPAEGIDGKGTPAYTRPIINFDLARVKPTGDGSRFGGNENYFLDFFIP
ASVLFGALGITSQTPLQFIFFSSANNNNFNKDFLIPVGSTFTSGFSDPVAPQGFNAQARLVLAKTLTSSPGAAVLTGQVA
SYTGTFAISNQGKSQANAVLVTDVFGLDTVSSLTVSGVTQGLSTVNPLSSTISWNAGNLAPGQTVTLTFIVQGAFLTAGA
GSRILNTAAGTGFDSLTGQAITAGPAANSIAVVTAGGVTGTVYNGMTALPLAGAALQLTNAQNQAPVNQTSGTGGAYGFS
NLAPGTYTLTVSFAGFAAQTVPVTVNAGTVTPLDITLRPQPASLSGTVADQAGNVLPGAVIRIQSPFGAIVAETTAGAAG
QYQLSVPAGTYTVIATANGFQSQVNTLQLQPAQSVSASFNLAALPGTVSGVITVSGSGTPLAGALVEALNERGQQIATAV
SAADGSYAISSLAPGNYRIRASEPSFTTDIAGAAVLAGQTATVNIGLSPAPGTLTGQVTDALSGVPLSGAVVRVVTAEGI
TAATAVTDAGGTYVINSLQPVNYTVTFSETAYASQTTGVSIAPGAQSVLDIALQQLAGSITGVVTDLNGNPLENAVIRIF
MNSFVVATLNADQNGRFDIHGLAPGSYMVRAETPGYQRQLIGAFIQPGEIVEVDFALAPQPGTLTGTVTTGTSPVQGAVL
VVRQTDNSVIGRTITSADGTFVLEELAPGTYTAAVFADGLQTQIKGFSITPDQTTTVSYNLEPLPGILAGTVYNASTNQL
LADIPVAVRIVDTNGLVIQSLFTDQFGAFTAPNLAPGSYTAIVSSTDFQTNLVSFLIAAGQTTTIRAGLIPSPGQISGTV
TAAGTDTGISGASIRVSNYTGTLIATVLTDSSGYFRAEGLAPDTYIVTASQTGFQQGSIGAIVSAGQTTTTGLSLLPLPG
SVSGAIFPAVDGTVIQIYDRNNLLIEEELVSTGNGTFRFDSLAPGAYIVTATPPGYAVAKAGAVVEAGQESSVTLTLSPL
LSVLTGRITDTADSPVDTAVIRVLDGNETLLGTGYADANGAYVVDNIPAGNHFLVVSAPGFSTVVLGIAALPSVDLTGID
IRLVPDSGSVSGQITNASDGSPLPSATVLIRSADVSNLLIATATSSLFGNFFVSGLSPGAYTVVAEAEGFAAGTAGAIVV
SDQETAASLALRPLSGVVTGSITDVSGATLTGSETQVKIFNEAGILLFAYLADADGRFTTPELAPGSYVLTASAPGFASA
SVLATVTAGSADTVTLVLTQQNSLLTGNVINRNGSAPVPGSMVTLLNAAGQTVDQSVADLSGNFTIDKVTAGAYTLSVTG
DGFGSTSLGVVLRPGQTTAVTVPLEPSPGSVSGFVIDRVTSAPLAGADVRIEDAAGVPVNTVQTDSSGFYLFNNLSSGDY
KVTAEAAGFGSQNGGAPVAPGRSTDYSFALSPLPASLSGTVTSAAGARPLAGVFVEVRQLNTYGTVLTDTTTDAAGRYTL
DGLAAGNYTVVFSLAGFGQQTDAFFAAPGQNTGVDARLSPTSNDLNGTVVTPPTVPVFPPPPVVDVYDDRNTPVVTVPTD
PSGQFTADGIAAGDYTVIVQKPGLSYSQQVVTVVPGFTPNIVLSPGSPPFAVTGAVQDESTGNPVPGASVQILNENLTPV
DRTLADGTGLFLLGGLRPGSYTVTVSSELFGSKSAAVSLAAGTPPPPVAVALNPLFGTLTGTVTDRNGRLLANVFVQLFD
PSGLFLRSVTTSTSGVYTVTNLAPGFYSARYTLEDKQPAVRTLIIAANETTVLNVVLEDDTEE
>lcl|NZ_CP026520.1_prot_WP_042230200.1_4657 [locus_tag=PC41400_RS23485] [protein=cation diffusion facilitator family transporter] [protein_id=WP_042230200.1] [location=complement(5189306..5190292)] [gbkey=CDS]
MSPDAERTNWLELIKKGNMSSATAALGNTIIAIVKGVAASLTGSGAMFASTMHSIADAVNQMFVFTGSVLAEKKATRRFP
TGFGRVINIFCMIAVLVVTFMAYETIREGLHLLQQPAESHGFWLNLSVLILSIVIDGYVLYKAMKEVIKESRAEASGFGI
VKASFKHVKRAAPPTRLVFYEDLVATTGALLALIAVVVTSLTNFSALDGLMTVLIGLLMIFVAFRVGYDNMVGLIGVSAP
PDVEDKVSALIFAESLVTDIYQLRVLQEGRYYHVEGLIELKPGLTLAEADDIKFRVRDKLLGDPDIADVTLGILEDNGVK
DWVPAADR
>lcl|NZ_CP026520.1_prot_WP_042230201.1_4658 [locus_tag=PC41400_RS23490] [protein=hypothetical protein] [protein_id=WP_042230201.1] [location=complement(5190492..5190782)] [gbkey=CDS]
MSRQTESGRQADTPAKTGPGARGLTDTRGGEIRQSAFEDVTAGGKSADDTIPLKIFEDVSAHERIGLRADDSAADGEVDL
DPSGAIQSKERQNVWS
>lcl|NZ_CP026520.1_prot_WP_042230202.1_4659 [locus_tag=PC41400_RS23495] [protein=carbohydrate ABC transporter permease] [protein_id=WP_042230202.1] [location=complement(5190893..5191792)] [gbkey=CDS]
MSKYDWAAEHAGFRRRGSEQAVKRTARYRIRASAFSAYFLLGAGAIVMLFPFLWMIATTLKTPQDVFSLSLIPPEATLDN
VRTLFAETLYATWIVNSLVVAVITTVSVCFFDTVVGYILAKFTFPGKTVIFVGILSTLMVPTEMLIIPWYLLAAKLELVD
TYLGVLFPGLISAFGIFLMKQFMESVPRDLLDAARIDGMGEWGILLRVVVPLVKPALATLCILTFLGSWNAFIWPLIVTQ
TPEHLTIPVGLAFFSSESGDSGSWTLIMAGAAISILPLLAVFLLFQKQIIRGIAMTGFK
>lcl|NZ_CP026520.1_prot_WP_042230204.1_4660 [locus_tag=PC41400_RS23500] [protein=sugar ABC transporter permease] [protein_id=WP_042230204.1] [location=complement(5191806..5192675)] [gbkey=CDS]
MKRQRLFFVYACLLVPLLFFIGVRIVPIVYSFLVGFREWDLLSEEKPFVGFSNYVALYHDPVFLTSLRNTVFYVLLGVPG
QLAAGLAIALLLRRIIRFRAFFRTVYFLPYVTSVVAVSWVFRWIFMKNGIVNGLLLQLGLPSQLFLGSPDQAIFIVTAAM
IWQNLGFQMLIFLTGLENIPKEYEEAAAIDGAGAWQRFVHVTLPLLNPVLLFSVIIASISYLQSFTQILNMTSGGGPLNS
TKSVVVYIYELAFKQFNMGQATAATVVLFLLILLLSLLQLKTLNRKVEY
>lcl|NZ_CP026520.1_prot_WP_042230205.1_4661 [locus_tag=PC41400_RS23505] [protein=extracellular solute-binding protein] [protein_id=WP_042230205.1] [location=complement(5192672..5194081)] [gbkey=CDS]
MSSKSEKNNGKTEQRQKGKPQPAFFRLAVLTLTAALSAGLMAGCSGSRSASGDGAKDASGQQSGDAGKDEKVTIEYWQYQ
YPVKVDLINSLIKEFEKTHPNITVKQTNFPYDQYNEKVATLIPAGKGPDVINLYYGWLPKYTASGYLQPLPKESFPDSKI
KEEFFPFVEAAKIGGSYYAIPTAVRTLGLYYNKDLFAKAGLDPAKPPATWEELVDYSKKLTEKDAKGEFVVEGMAWQPNS
QLHHWYRDALVYQAGGKDISEDRRRVLWNETPAGLQAFKYLLDFATVHKVGTKDFYNDDITAFKGGHAAMNIDGSFQLGG
LKKDVPNLNFAVAPLPAYKGKATQASFWANGITKNAKGKKLEAATEFLKFLTSKEVQEQWVEKVGELPASKAVALQDKYA
KDEKIGPFIRQLGDANAHFFVDEKLERDYFVQAADSVVLNKVPAEKAFEELVAKTQKLYDDYWAKQAKK
>lcl|NZ_CP026520.1_prot_WP_042230206.1_4662 [locus_tag=PC41400_RS23510] [protein=amidohydrolase family protein] [protein_id=WP_042230206.1] [location=complement(5194078..5195028)] [gbkey=CDS]
MFKDGFKVFDIHGHLPYAQQFVKHDEVIDGAQAARGERMRLTWDFPSGPDTGPGAGRPLIDRWADELDKYGIGGLNFLTA
RGNDDLKEQISKYPGKFTGFAHHSIEAPDAVEELQRAVDELGLRGYKLFGPLIQTSLDDPSLRPVWSFLAERKLPVLIHF
GLLGHTGGIVQHRNINPLAIFNVAREFTDIPILIPHFGAGYYQELLHLCWGCPNVIVDTSGSNQWMRWAPYDLTLETLFR
KTVELIGPERVVYGSDSSGFPRGYVYRYLQDQVRVCRELRFRETDIQSIFGNNARRLLRLPDTAASVQEHKGVEVL
>lcl|NZ_CP026520.1_prot_WP_042230208.1_4663 [locus_tag=PC41400_RS23515] [protein=TrkH family potassium uptake protein] [protein_id=WP_042230208.1] [location=complement(5195280..5196635)] [gbkey=CDS]
MGKKRRILVRFTPPQILALGILAIILIGSVLLSLPAASSTGEPISYLDAFFMSTSAACVTGLSVLNAGVHFSLFGQIVIL
MLVQVGGLGFMTMATLIALVLRKRITLKERLLLQEAMNQGSVEGIVRLIRKVLTYALAIELAGAVLLAVRWAFDMPLGQA
VYFGVFHSISIFNNAGFDLFSSLPGAQGSLMAYVNDAYVNIVVIGLMILGGIGFIVIADLLDFKRTRKLTLHSKVVLTAT
GLLILLGSVVVFIFEYTNPLTMQPLSLPGKIFAAFFQSATTRSTGMNTIDIESFRQATQFFVIILMFIGAAPGSAGGGIK
ITTFAILLGAVITMVRGKEDIVFFRKRLPEERIYKATTLTLASLILVVTGTMILSVLQDGPFIKMLFEVTSAFGTAGISL
GLTPELTSPSKVLMAILMFVGRLGPLTLAYALTPKGTKELYRYPEGKITIG
>lcl|NZ_CP026520.1_prot_WP_042230210.1_4664 [locus_tag=PC41400_RS23520] [protein=hypothetical protein] [protein_id=WP_042230210.1] [location=complement(5196774..5197322)] [gbkey=CDS]
MKKVMLLGDSIRIGYQPLVRAALEGQAEVVGPEENGRFAKHTLWGANLWIRDLGTPDIIHWNNGLWDLHHEAPMVEALTS
LDEYILTLGRILNELRRTGAAVIFATTTPVPVDAIGRSNAEIDAYNAAAAELMRHHGVEINDLNAVVKRDLQSFICEDNL
HLSEAGNRACADRVIEAVRKYI
>lcl|NZ_CP026520.1_prot_WP_241688918.1_4665 [locus_tag=PC41400_RS23525] [protein=zinc-dependent alcohol dehydrogenase family protein] [protein_id=WP_241688918.1] [location=complement(5197396..5198334)] [gbkey=CDS]
MLEIRRTPVVPPGPGEIRVRMLARPINPSDLIPVRGAYAHRIALPAVPGYEGVGIVEETGPSVPPELLGRRVLPLRGEGT
WQESVLAPAGLAIPVPPDIPDETAAQLYINPVTAWAVCGALRLKYGDTLVVNAGGSAIGRVFAQLAAVFNYELIAVTRSE
AHTEELQRLGLAHVIVTAGTPLREAVLSKTGGRGATAAVDSIGGPDGAALAGCVRPGGTVFSIGLLSGVPADWAALSRLY
RVDARPFWLRHWIRHASTADWFGAFEEIMELVRDGRLSLMKIAAGFGLDEVHAAVAAFESGRAGKILLTGGR
>lcl|NZ_CP026520.1_prot_WP_241688927.1_4666 [locus_tag=PC41400_RS30440] [protein=hypothetical protein] [protein_id=WP_241688927.1] [location=5198215..5198577] [gbkey=CDS]
MGIGPAHGDQVRRIDGPGQHPHPDLARAGRDDGGAADLQHFSRRAEPVVIDAFVKHAVLSRDVNFQKPGQGRHTLNGSAA
NVRTGFLPRAAPGPPNGTVHPPTAAPGLRPFRRALSGLSE
>lcl|NZ_CP026520.1_prot_WP_042230212.1_4667 [locus_tag=PC41400_RS23530] [protein=hypothetical protein] [protein_id=WP_042230212.1] [location=complement(5198536..5199225)] [gbkey=CDS]
MKIVELTLHASNLSELKQFYGERLRLPLTGETGASFSFRAGNTNVTFVRTAQDDDSFYHFAFNIPENKIEEAADWLRDRV
ELLGEEGDPVVHFESWNAHAVYFKDPAGNIVELIARHNLDNASQKPFGEEDFLCVSEIGMPVDDVSAALRALKREADLEA
WMTPSDTFAPVGDEHGLFIVVKKGRVWFAGEKSSHPHPLSVRTDGSLVRLDDREGISFGEPGESTPERP
>lcl|NZ_CP026520.1_prot_WP_042230214.1_4668 [locus_tag=PC41400_RS23535] [protein=GNAT family N-acetyltransferase] [protein_id=WP_042230214.1] [location=complement(5199302..5199796)] [gbkey=CDS]
MIRICTEKDIRAIYDIINDAAAAYKGIIPADRFHEPYMTMEELQSEIGSGVVFWGYEDGGRLIGVMGIQDKGDVFLIRHA
YVRTDQRKGGIGSRLLAYLTNLTEKPFLIGTWADASWAIAFYLKNGFSLVPEAEKERLLRTYWTIPDRQIETSVVLGDRR
REAV
>lcl|NZ_CP026520.1_prot_WP_042230216.1_4669 [locus_tag=PC41400_RS23540] [protein=hypothetical protein] [protein_id=WP_042230216.1] [location=complement(5199944..5200393)] [gbkey=CDS]
MNAFLKLNGISAGYACVVLVFGHGLLYGMNVVKPLGLSYSISVKMVVGGTFLLAGLLTWFCMRITKNWMQGRASAFWAAV
LWFPYWLLIGAVLALVLPGEDNFYAMYMLAVFLIPFYPLFTATAIGISALWHESAKKRQNRRTAGTNIP
>lcl|NZ_CP026520.1_prot_WP_042230218.1_4670 [locus_tag=PC41400_RS23545] [protein=RidA family protein] [protein_id=WP_042230218.1] [location=complement(5200509..5200919)] [gbkey=CDS]
MTKRKNISSGSPWEPVVGYSRAVRVGNIVEVAGTTAMKDGAVVGIGDAYAQTKCALEIIEEALNRAGAKLADVVRTRMFV
TDIGRWEEIGKAHGEFFKDIRPASTMVEVRALIDPELLVEIEVQAIVKETSSGFGD
>lcl|NZ_CP026520.1_prot_WP_042230221.1_4671 [locus_tag=PC41400_RS23550] [protein=hypothetical protein] [protein_id=WP_042230221.1] [location=complement(5200986..5201555)] [gbkey=CDS]
MLFSKKDPADPRTDFERDLLRQARRLTPGEPPPPWRLQTVIEAAGVTAGGWDGDGRIVLIGSGYSITDAFTGQRLVRERD
EAKTEGHLDAGGLKFAVPETGEEIGVFGLDGGDGIQMNAEGWRLEVIYPWWPRASVILENVFSREYRYLAGAVMLEIPRL
DGWLKGGFSPSGEAFMVLGSGGAAVFAKR
>lcl|NZ_CP026520.1_prot_WP_042230223.1_4672 [locus_tag=PC41400_RS23555] [protein=DUF1361 domain-containing protein] [protein_id=WP_042230223.1] [location=complement(5201610..5202296)] [gbkey=CDS]
MIHLSAWKSAAFLAALSALTLLVYSVFRNKTYFDFLTWNLFLAWVPFVFSFIAYEIGRRAQSRMRAAIIVVLGAGWLLFY
PNAPYIVTDLVHLTAMKGLYASGQGTFAFRYWYDLLVLVLFAWNGLLLCVYSLYQVHSLIRLAAGSLLAWGFVAAAALLG
GYGVLLGRVYRLNSWDVLRDVSLIKRIALESLHAPVMLFSLLFGLMIGIVYVSFYFVVHQAVRAAKRA
>lcl|NZ_CP026520.1_prot_WP_042230224.1_4673 [locus_tag=PC41400_RS23560] [protein=GNAT family N-acetyltransferase] [protein_id=WP_042230224.1] [location=complement(5202342..5203118)] [gbkey=CDS]
MHTMTKVNAAIRPHLTRLFEEVRRVDVCIDSVLEGQAGGEIRIYVDSLTNPSSAQINQGSFVYFAGDSTCPSAGQWLEEL
PAHCYIMPSSPGWVQRARHIHGGKLIRGERYMFSPESLRPGPLKELTEHPDYKVEKITPELARDMFGDPLQHYHFVNYDT
AEQFMETGFGYCIRAEGEVAAACTTAMVCRTGVEIGIMTHPDYRKFGLATRVAAAFLLHCLEHDLYPSWDAANLKSKGLA
EKLGLIYAGSYEVFSLQD
>lcl|NZ_CP026520.1_prot_WP_042230227.1_4674 [locus_tag=PC41400_RS23565] [protein=DUF1572 family protein] [protein_id=WP_042230227.1] [location=complement(5203178..5203774)] [gbkey=CDS]
MSANQIYLESVRKQFLYYKTLGEKAMDQLEPEQLFTAYNEDTNSIATIVAHLSGNMLSRWTDFLTTDGEKEWRDRDAEFG
HSLSDKEQLLDKWKEGWNCLFHALDSLSPDQLEQIIYIRNEGHTVIEAINRQLAHYPYHVGQIVYAAKLLKKEEWNSLSI
PRNKSAQFNSEKFAEQKSVRNFTDEEMKRLSANTDSMQ
>lcl|NZ_CP026520.1_prot_WP_164977151.1_4675 [locus_tag=PC41400_RS29965] [protein=hypothetical protein] [protein_id=WP_164977151.1] [location=complement(5203789..5203947)] [gbkey=CDS]
METFGRGCLYIILGIVVLMVFALFAGGTVTIPWIIAIPPVVPAFWTASKKNK
>lcl|NZ_CP026520.1_prot_WP_042230229.1_4676 [locus_tag=PC41400_RS23570] [protein=GrpB family protein] [protein_id=WP_042230229.1] [location=complement(5204009..5204533)] [gbkey=CDS]
MVNKTKEVRIVPYDPAWKIAFESMRAELERHIGDLVLAIEHVGSTSVEGLGAKPILDIDVVMDSYEVFPAIVERLGKAGF
RHEGNLGVEGREAFKRTYEDGFMAYHLYVCPKDGRGYLEHIAFRDYLRAHPAARERYQRLKEDLAAEFRYDRDTYCERKT
GFVRGILNKTLYKD
>lcl|NZ_CP026520.1_prot_WP_042230230.1_4677 [locus_tag=PC41400_RS23575] [protein=GNAT family N-acetyltransferase] [protein_id=WP_042230230.1] [location=complement(5204527..5204940)] [gbkey=CDS]
MNIVYSDEIPSGEAIYDLYEQFGWNEFLKLSARQLHDTMRGSWYVASAYDGGRLIGTGRVVSDGVMTAFLCGLAVHPDYQ
KKGIGSELMKRLMQRCITRQLHMELFCSDPIVGYYERFGFEVFASGMKYRKEGQCPW
>lcl|NZ_CP026520.1_prot_WP_042230233.1_4678 [locus_tag=PC41400_RS23580] [protein=DUF962 domain-containing protein] [protein_id=WP_042230233.1] [location=complement(5204993..5205322)] [gbkey=CDS]
MSMERVRRDMKGYLEEHRNPVNRTLHYFAFLAAFMAWIFLWIDIRLTLALAVLHYALAWTGHFYFEGNKPASFRYPLVGF
YAGFLWFFLRTAEMATRTDLVDAWVSKSE
>lcl|NZ_CP026520.1_prot_WP_042230235.1_4679 [locus_tag=PC41400_RS23585] [protein=alkaline phosphatase] [protein_id=WP_042230235.1] [location=complement(5205539..5207170)] [gbkey=CDS]
MAKKMLGMLLTSVLVCGSLGAGTGHAQSGRALPAEKKEVKNVILFITDGMSLSDVNLTRWVQGGKPLEMDAYFKGLVRTY
ASDSLTTDSAAGASAYATGHKVKSETVSISPDKVTMPFIDKPSEQEAANPLPTLLEAARLKGKGTGLVFTCELTDATPAV
FASHAESRDNAHSIAEQMVYSGVDLVLGGGSGYLVPGDKEINRKDGEDLTKVLKANGYEYATSRIDLLSSQTNKIWGLFN
EGALDADLDLDPQKQPTLAEMTTIAANKLSSNQNGFFLMVEASQIDWFGHDNDPVGIVTETQAFDRALKAAIDFAKKDGN
TAVISVSDHATGGLNMTNYDTMKDLQPIFTKAKHTSYGLEGRINESNYKKMLEEEYGLSDLTKEEEEAVKKGLKDNLSPV
IGGILGERAGVSFSTGDHTSEEVGLFAYHPNDFLPTDYAGRGVVENTDVGKYLQLILGLDLPALENKLFLPASDLEAKGA
KVTVDKSDKENPVVVVTKGTKTIKLPVDKNIALTDGKTIKLSTPTLLIKNKVWVSSDAAALIQ
>lcl|NZ_CP026520.1_prot_WP_042230236.1_4680 [locus_tag=PC41400_RS23590] [protein=helix-turn-helix domain-containing protein] [protein_id=WP_042230236.1] [location=5207397..5208236] [gbkey=CDS]
MKNVCYDCDSLHISRKELAYTSMPSRHTHDAFEMMYLVSGQLYYFIGDKTYQVVGGVLLFINANVPHQLVNAGDAVFERV
TLLFKEEFLHDFFRLQDGGELLSFFSADFHAVHLSGYEQTFAEELFHKMLHEGVHKQIGFEQYQKVLLMELLLFMNRKVR
EADREQAGSGASPTHKKIYALVEYINNRFYERLTLEDLSHTFEISPSHLSRTFKEVTGYTAIEYINNVRIKKACALLRET
SWKVLEICERVGFDSLTHFGRTFKQMTGCSPVRYRRRQP
>lcl|NZ_CP026520.1_prot_WP_042230240.1_4681 [locus_tag=PC41400_RS23595] [protein=GNAT family N-acetyltransferase] [protein_id=WP_042230240.1] [location=complement(5208421..5208990)] [gbkey=CDS]
MNILVKPLDNEELKEIWMHAFRKHGLHREAAYYDQCLHENVLGTRVTLLAFVEETLVGCGHLKVESEYAYFRDRRIPEIN
DLNVFPDYRRSGIGTKLLERFEALASEDFHQIGLGVGLYKDYGAAQRIYCRRGYVPDGNGLMSGNREAAAGEMVRVDDDL
LLYLVKDLRLDSFYNLHAGKAEIQPFLNH
>lcl|NZ_CP026520.1_prot_WP_042230243.1_4682 [locus_tag=PC41400_RS23600] [protein=histidine phosphatase family protein] [protein_id=WP_042230243.1] [location=complement(5209028..5209600)] [gbkey=CDS]
MTTIGFIRHGSTHWNKEGRAQGSSDIPLDEDGLAEAEKLAERIGSEKWDSIYSSHLLRAKQTADRIGGRAGIPVWSDERI
REAGGGEIEGTTEQERIAKWGAGWRELDLGIEKPESVIARGRSFLDEVLSRHQGKRILIVSHGAFIRTMLNHLLPDSEYD
GHLTNTSVTTLVLDDRDWTCGLYNCTKHLE
>lcl|NZ_CP026520.1_prot_WP_042230246.1_4683 [locus_tag=PC41400_RS23605] [protein=hypothetical protein] [protein_id=WP_042230246.1] [location=complement(5209631..5210029)] [gbkey=CDS]
MYSIKSPGGSLFPYYYKGGEIHCLKYGSFFQDEEGLFRLMLEEEQFIGQRSQELRIWVDFYETRLTDDVITAFVDHIGRL
NTHVYKLTLVGCSFGSKLKLKKRLNRLDPPLKRPVSFFSDPEVAKTWLVREG
>lcl|NZ_CP026520.1_prot_WP_042230248.1_4684 [locus_tag=PC41400_RS23610] [protein=DinB family protein] [protein_id=WP_042230248.1] [location=complement(5210203..5210724)] [gbkey=CDS]
MLIKPPLGEYPAHYETYVKLVQEGNLAEILVEQLQTTTELIQSLTEETGDHKYAPGKWSIKEVLGHVADTERIMAYRMLC
IARGDQTPLPGYDENEYVAGASFAASGLSDILKDYSAVRCSTLTLLRGITEEAWLRTGTANNGLISARALAAIIAGHERH
HIHIIQERYLSTV
>lcl|NZ_CP026520.1_prot_WP_042230249.1_4685 [locus_tag=PC41400_RS23615] [protein=hypothetical protein] [protein_id=WP_042230249.1] [location=complement(5210875..5211078)] [gbkey=CDS]
MTNNVINTVLVVVLAIVSFFTGEVVTFIMLGFILMSLITVTAYLRKIYETLSRMASEKDRPPGPPGE
>lcl|NZ_CP026520.1_prot_WP_042230251.1_4686 [locus_tag=PC41400_RS23620] [protein=tetratricopeptide repeat protein] [protein_id=WP_042230251.1] [location=complement(5211127..5211609)] [gbkey=CDS]
MTKLEEAIRLREAGRHAEAQTLLLDLVREEPLVPSNWYQCAWIHDAMGLEKEAAPFYEKALELGLADEERRGAWLGLGST
YRTLGEYESARACFRQAIREYPEAREFQVFYAMTLYNLGEHALAMELMLRQLGETSGDEGIASYKKAILYYADKLDRIWK
>lcl|NZ_CP026520.1_prot_WP_042230253.1_4687 [locus_tag=PC41400_RS23625] [protein=GNAT family N-acetyltransferase] [protein_id=WP_042230253.1] [location=complement(5211647..5212126)] [gbkey=CDS]
MTEHDAFVTISGDVVYRQAVVSDAPGMARLSAQLGYPTGETRLAERLGHILTLPDHVILTAEYEGGVIGWIHAHARCLLE
SEPFAEIAGLIVDRKFRGRGIGRTLVKRCGEWTSSASLGRLRVRMNVNRQEAVSFYEQMGFEPVKTQRVFDKSIRPRLK
>lcl|NZ_CP026520.1_prot_WP_042230254.1_4688 [locus_tag=PC41400_RS23630] [protein=GNAT family N-acetyltransferase] [protein_id=WP_042230254.1] [location=complement(5212138..5212647)] [gbkey=CDS]
MTHGVTIHELESMEAGVLNDLTELLIRVVEDGASIGFLPPVSWEDANTYWKKVIEPGVLAWNAVCDGKIVGTVQLHLAMK
RNATHRAEVVKLMVHPEYRRKGVAHMLMQALEKTAAEEGRSLLVLDTREGDASNKLYKSFGYVEAGRIPHFAKSSDGNLH
TTIFYYKLI
>lcl|NZ_CP026520.1_prot_WP_042230255.1_4689 [locus_tag=PC41400_RS23635] [protein=phosphotransferase] [protein_id=WP_042230255.1] [location=complement(5212714..5213727)] [gbkey=CDS]
MNEKVPLSAILAAYPVREPSIRFIRHNENMTYRVTDGTTGQSYLLRIHKPVTDKLHGLQHSREGLHAEMQWLADLGAAAG
VRVQKPVASHEGEWVTSVNLDAESGTQSGVRSGIFCTLLEWIEGRDVGQGESFSRETVTGLGRQLGAMHRFSQTYPGAAI
MKRPRYAGAEQNEEMLTRIRSGLSMDLFREEDLEIVGQTFTRIADKLERHSRSGSSWGLIHADINKGNLIVTDQGLSLID
FCLFGHGYYGYDLAGCALTMKTELRDSVLEGYSLVRPGAVTERLDLLEGFMLLSIFGFYSFHLHNPDKHPWMKERMPAFC
KNYCVPFLEDRSLFYTM
>lcl|NZ_CP026520.1_prot_WP_042230257.1_4690 [locus_tag=PC41400_RS23640] [protein=hypothetical protein] [protein_id=WP_042230257.1] [location=complement(5213905..5214360)] [gbkey=CDS]
MGFFFPQGSFTQPQESQGTQDVQRSFGTMGPFFPSPGGGPGQGPFFPFPGQQGPGMLPPGPPGFPGGSPGQQAPTAPPPQ
FEPQVAPFAVDPGGIVGCLFRNTFIWLNDGTRFWYFPTFVGRTSVAGFRWTGFSWVYFGIDLRRISSYTCF
>lcl|NZ_CP026520.1_prot_WP_042230259.1_4691 [locus_tag=PC41400_RS23645] [protein=serine hydrolase domain-containing protein] [protein_id=WP_042230259.1] [location=complement(5214446..5215474)] [gbkey=CDS]
MDNEQSLELHRSDAFSPLNQYVLGVQEKIAASAAAVYVLHRRTVVNEWYSGTHGFSPGSRPVGADSRFNVASVRKTYLGF
AVSLALYEGRIGGLDDPVSDYIADPDGALLAGTTIRHLLTHTHGLASPQKRIFPPGAGWAYNNVGVNLLIRMIGEVFGKP
LAQLLNEKVFRPGGFTETGWRKHQTDGLVWLNEAYSSEEGTDINLFVSARELASWGQLHLSKGMWNGMQLIPRPVWEQTA
EIISPEGLGDALPRNGFFWFVQDRPRSLTELGHELPAGSYQSLGSTGCACLVIPEKEAVAVRMYNQTGPNPGGYSYLDDI
RTFGNTVYGCLNVMDLQADRSP
>lcl|NZ_CP026520.1_prot_WP_042230260.1_4692 [locus_tag=PC41400_RS23650] [protein=hypothetical protein] [protein_id=WP_042230260.1] [location=5215658..5216026] [gbkey=CDS]
MRKVFRIYGLTALILVMTATTAYAANQESTAYATSGGYDANSSYIDGQGQFGRIKFNSSGGGWGFYEQTCSGSGTTYGPA
LYIPADGGSINELVAMPSNCKFKLHITSGKYFPVSGTIQNYE
>lcl|NZ_CP026520.1_prot_WP_231392012.1_4693 [locus_tag=PC41400_RS23655] [protein=YHYH domain-containing protein] [protein_id=WP_231392012.1] [location=complement(5216208..5216384)] [gbkey=CDS]
MRTLKKTVIFIVAIVVVMATSASAYAHPGRLDKNGGHNCSAKSKQKGLCSGYHYHKKK
>lcl|NZ_CP026520.1_prot_WP_042230263.1_4694 [locus_tag=PC41400_RS23660] [protein=M60 family metallopeptidase] [protein_id=WP_042230263.1] [location=complement(5216584..5218800)] [gbkey=CDS]
MRKLKKSLQILSLMAVTAATVVVPVSPPRVLAQTSETAITQQSNLVVQRNFEVKALGDIWAEGERERRAKGYADARKIFQ
PTGLYAKANEQIEIEVSGDKPITAIIGTHRHDKEWAIRYPLKAGKNTISSPNGGLLSFDNSNNEGSIQVNVVSGGSPVPF
FVLGKNTKADWQAMMSAYPNAPSVVLQSEKALLVFTYASAQNHILNQDPVPVLQTYDTLIGAQDKISGLSNSDSDPRHHL
DKHLIGFIESPLDTGSYYAMATWDGAIMPTRSEANADALNLTRMGWGQYHEAGHLRQQEPWTWGEMSEVTVNLYSIAAKK
ALKPTEPARSQGEYLTAFTFVDQPTKNFTDSNSKLEMLWQLNLAFGDNFYPELHRLYREMAKTDLPTTDDQKKQAFILNT
SKVAHYNLTPFFDKWGLPATNETKNKINSLSLPLLTAPIWFGSEYNIIKPTDGVDKVKGIGGIIAATTNSQETSGENNAA
AYAFDGNTNTLWHSEWSKPNQFPYNITAKYLNPSTFFKLTYLPRQTQENGIITKYRILTSLDGVSFTEVATGTWAKDKTE
KTVTFPPTLAKYVRLEVPQGGGTNGFASASEIKIFETAQPVTTYLSDIDWFSATIGWGTIRKDRSVDGNTLKLNDKAYTK
GLGTHANSEIVYKLDGKYTSFAALVGVDSEVGSVSTVEFQVLVDNQVVFSSGKMNKTTEAKEVNVNLSGKNELKLKVTDG
GDGIDSDHADWVNARLIK
>lcl|NZ_CP026520.1_prot_WP_042230265.1_4695 [locus_tag=PC41400_RS23665] [protein=stalk domain-containing protein] [protein_id=WP_042230265.1] [location=complement(5219240..5219959)] [gbkey=CDS]
MKKKYMALLGACGLAAVLGATSYAAGGVEKIEAYLNPDIKITLNNEAWTAKNGEGKALAPVIIDGTSYVPAKAVVEAMGG
QVQWNNDTKTIAITSVPGKEAPVFTPQSGETDRDKQIAQKIAEIKEKLKLGLTREEVAARLQVKLENATDNGDVENGSDS
FSKVDYFKEAGYERPEGTPDHVIDYEGLNNKKVGASLFIGWKNDKLHFYSISYVNPADKKVHLFVVGLDGTSEDSPAAN
>lcl|NZ_CP026520.1_prot_WP_042230267.1_4696 [locus_tag=PC41400_RS23670] [protein=DJ-1/PfpI family protein] [protein_id=WP_042230267.1] [location=complement(5220106..5220687)] [gbkey=CDS]
MKIAFIVFDGITFLDFIGFYDVVNRLHLFDKTKGTTWDICALAPEVKDEQGLTIRAGKIKPDLSEYDLLFVPGGLGTRKL
RFQTEFVDWLRQGLEVKHVVSVCTGSLLLGAAGFLENRKATTHPKAYELLEPYCKEVVRTRIVKDGHVVTGGGVSASIDI
GLYVVEMLAGREAADQVKEQIDYPYERIGMIEV
>lcl|NZ_CP026520.1_prot_WP_129112530.1_4697 [locus_tag=PC41400_RS23675] [protein=DUF5780 domain-containing protein] [protein_id=WP_129112530.1] [location=5221064..5222260] [gbkey=CDS]
MFKCSKCGNETLEEDSLFCKKCGSKLSSDSTKPDKRKKFWIGGSLASLLLIGVAVTIIILNNPMTKFKRAVQGNQVSEAT
SVYNQNIKGDAEKEKKADAFITDEVTRIQEEFKNNKLEHQQAIASLETLKKLEVLKPSVEAAVNNVNQLNESRTSFKKGQ
EYLANKNVKDAIAELKKVIKEDQNYAQAQELISTHTTEYKTAALKDAEQFSDSQDYDKAIGILSDALVVAPNDSDMIAKK
SSFQKLNEAKKAEERKKKIEETKSSQEVSADHANIVIQSTEYKALYPDMIQVVVRNNTDKTVKSMSVSMLGYDSNGFPVK
IEFYYGGSASFEFIGSAESVNIIPKGTYGKNRGWKIREGHGIKTVLAAVKEVEYYDGTKWTNPYYEYWLDENKEKPLH
>lcl|NZ_CP026520.1_prot_WP_042230270.1_4698 [locus_tag=PC41400_RS23680] [protein=isochorismatase family cysteine hydrolase] [protein_id=WP_042230270.1] [location=complement(5222344..5223021)] [gbkey=CDS]
MNTYIQPNLVTAALLTIDTQNDFSLSGAPAEIPGTSEVLPVMGRLLEQCRAHGIPVIHVIRIYERDGSNADLCRKAMIEA
GAAIAAPGTWGADLADAIKPPHAEELRFEALLGGEIQAIGEREWVIYKSRWGAFYNTQLEDFLKERGIDTLIFTGCNFPN
CPRTSMYEASERDFKAIMVEDAISGVYEKGKEEIRNIGISIYDSASLIKDMNQAFAQSQGLPASR
>lcl|NZ_CP026520.1_prot_WP_042230272.1_4699 [locus_tag=PC41400_RS23685] [protein=DUF418 domain-containing protein] [protein_id=WP_042230272.1] [location=complement(5223106..5224185)] [gbkey=CDS]
MVVAKKEARVSGLDLARALSIFGMIVVNYKIAMMADGSGPSWLAGFAGLFEGRASAVFVVLAGIGISLMTRKARRGDEPE
LAKQKLVLRKRSAYLLALGLFLLALGWTADILHYYALFLFAASFLIRSSNRFLTGAAVIAVLAGQCAQLALNYGAGWDAA
FHSYTDFWTVGGFIRNLFFNGFHPALPWFAFFLTGMLIGRLDLTDPSVRAKRVTVSLIVFAACELISRLAMKLSSAAVGD
AGGRYLFETKPMPPNLLYMISATSTAVLVIFACIYVMEKLPGRRMLTALVHTGQMALSHYILHVVAGLGLLEALGMLENG
SLPFSVAYSIGYFAAAVGASLLWRRKFTRGPLEALMRKL
>lcl|NZ_CP026520.1_prot_WP_042230275.1_4700 [locus_tag=PC41400_RS23690] [protein=glyoxalase superfamily protein] [protein_id=WP_042230275.1] [location=complement(5224289..5224645)] [gbkey=CDS]
MTLTGITPILRIFDEKKAKQFYVEFLEFTLDWEHRFEPGMPLYMQISRDEIVLHLSEHYGDCSPGAALRIAVSGVEDLQK
TLIGKHYPYARPGIERTPWETKECTVTDPFGNKLIFFE
>lcl|NZ_CP026520.1_prot_WP_042230276.1_4701 [locus_tag=PC41400_RS23695] [protein=YdcF family protein] [protein_id=WP_042230276.1] [location=complement(5224689..5225429)] [gbkey=CDS]
MKRWSGIRKAKVADVLLVLTFLSSPFLFRGRFGMMLFGIAVVLLLIIRRIDFSKRVRLRKTLWAGSLLIALSFAVVESFV
LSQMGADEAGTSGADTVVILGSGLKGYEVSLTLQQRLDAGFAYVSGKKDITVVVSGGQGPGENVPEAVAMRDYLVKKGIA
ENRILLESRSTSTLENLRFTKAVLKEKGINDPKILIVTSDYHMYRAKLLAKREGYEAYGLSSPSPAHLKPINMIREYFAM
IKTWVS
>lcl|NZ_CP026520.1_prot_WP_042230277.1_4702 [locus_tag=PC41400_RS23700] [protein=VOC family protein] [protein_id=WP_042230277.1] [location=complement(5225500..5225850)] [gbkey=CDS]
MKSPIQNKILTVFIPVSDIEQAASWYCKILGLPADGEVLFGHLYILPMQGAGIVLDSKIYSPETVFKAPVVQFATDDIEQ
SYAYMKSLGVRLVTGIENGHWFNFTDPDGNLLMVCK
>lcl|NZ_CP026520.1_prot_WP_042230280.1_4703 [locus_tag=PC41400_RS23705] [protein=HXXEE domain-containing protein] [protein_id=WP_042230280.1] [location=complement(5225922..5226452)] [gbkey=CDS]
MLFDDYFSLNAVVWLFLAAFMIHDFEEIIFVEPWMNRNYERLRPLIPGPGQKLFKDMSNVKSSQFAVAVFVEFILFVPVT
YLAAEREIWALFIGFNALLFVHVFTHLGQSILLRSYTPGVVTAVAISLPYSVYLFYRLLRDGAATWQDLLTYAPLGLLLL
PVVLAGHKLGKKVVPD
>lcl|NZ_CP026520.1_prot_WP_042230281.1_4704 [locus_tag=PC41400_RS23710] [protein=metallophosphoesterase family protein] [protein_id=WP_042230281.1] [location=complement(5226472..5227203)] [gbkey=CDS]
MDKIAILSDIHGNIPALEAVLEDIRRKDISRIFCLGDLIGKGPHADQAVDLIRKHCEEVVSGNWDDFITRETEIEDLKWH
QNVLGKERLEYLAGLPFSLEFYMSGKYVRLFHASPRSLYERVQPWDEPEKRLSLFESSEKCRERIPADIVGYGDIHNAYL
QHLYGKVLFNVGSVGNPLDLTQASYVVLEGSIDCRTTAPLEIRFVRVPYDIERSVSQAVESGMPALDAYIKELRTARYRG
IKD
>lcl|NZ_CP026520.1_prot_WP_241688724.1_4705 [locus_tag=PC41400_RS23715] [protein=methyl-accepting chemotaxis protein] [protein_id=WP_241688724.1] [location=complement(5227422..5229515)] [gbkey=CDS]
MMLKSKTKRIRGIFGKINLTGLFAKIMLISVACMTIPMLIALWYASSTSADSLEGEARKSMTNIVSEKVNQMDLAFTNLS
LSSSMIANNPFLVNTFKEIKKTGNPKSENAATASAYLGKLITDANGLYENIYAAYNSAIVSDGIGGQSVGMDIGVADLDK
SLLEGMKKGPVLFGPVQSPVTGGPIMALLSHIPDGSTDAMPSIVGTAVDLTKLSANIVRSTEEGNVKTFMINAAGVLVAS
EDGKEVLKYDMTKAEGDVPAFFKEVSANKKGVGYFTLNGQQQIASYAKSSLQDMFVISFKPVDEYMQDVTALKKGLVIVI
IGSILVFAAILVLLSYRITTPIKVATGHLKVIAGGDFSHPVPQKFMKSKDETGVLMQSMHTMQASISDTIRTVVEESDKL
GSSVTAVSSHLTDLNGQIQNVSGTTQQMAAAMEETAASTEEMNHSSSSIRQSVDRIADKAKQGEEASKEVSKRAEDLRET
AVASSERAVEIGRQMRESLESAIEQSRAVEQIQVLANSILEITAQTNLLALNANIEAARAGEAGRGFAVVAGEIRKLAEV
SGHSANKIREVVQIVTSSVQNLKANSESMLTFIDTTVINDYNAMVETGQQYYQDAEFYEKLLGDFNVTAHELSRAIEDMA
LMINEISIANNESAAGTGSISDQASVAMDSSDKVLNIVEGTRESAEQLKQTIANFKI
>lcl|NZ_CP026520.1_prot_WP_042230282.1_4706 [locus_tag=PC41400_RS23720] [protein=DUF2569 domain-containing protein] [protein_id=WP_042230282.1] [location=complement(5229798..5230313)] [gbkey=CDS]
MEPFVRDLNQNGKEVYEDDYTRPGISGFGGWLIVFSIGLILTFIYAVYSLVDYVWPMVESGEVADYFSYYPALGAAVLVE
TVSHLFYAVCPLVLGYLCFKQKRLFKTMSIAYLLLSLLLSTVAYFAYLSVSELASDFVDGALRDLIKNGITCLIWTLYFV
KSERVRNTYVV
>lcl|NZ_CP026520.1_prot_WP_053228711.1_4707 [locus_tag=PC41400_RS23725] [protein=helix-turn-helix transcriptional regulator] [protein_id=WP_053228711.1] [location=complement(5230490..5230909)] [gbkey=CDS]
MTTLGEVLKNLRGKESLREAGQRAGVSHSYLRYVEKGVDPRTGAALRPSPETLKRLAKAYNYPYQQLMNMAGYSGTETGE
TETNYLYLFDQETLERFDELSEGDKLFIVSQFESSVKYFYNNHLASKEAASGNTPVERT
>lcl|NZ_CP026520.1_prot_WP_009673339.1_4708 [gene=sigK] [locus_tag=PC41400_RS23730] [protein=RNA polymerase sporulation sigma factor SigK] [protein_id=WP_009673339.1] [location=complement(5231027..5231728)] [gbkey=CDS]
MPGLFSAIAFFIKELTLLVSYVKNNAFPQPLTEAEESKYLQKMAEGDAHARNLLIEHNLRLVAHIVKKFDNTGEDLEDLI
SIGTIGLIKAIESFSPTKGTKLATFAARCIENEILMHLRSLKKTRKDVSLHDPIGTDKEGNEITLIDILGTEADDVVDKV
QLKIEKSKIYRNLDILDEREKEVVVGRFGLEHGGEERTQREIAKELGISRSYVSRIEKRALMKLYHEFYKAKR
>lcl|NZ_CP026520.1_prot_WP_042230285.1_4709 [locus_tag=PC41400_RS23735] [protein=copper amine oxidase N-terminal domain-containing protein] [protein_id=WP_042230285.1] [location=5232091..5232693] [gbkey=CDS]
MKKKIIMLVAAFSVFAGVVSASSINGVYKGNAIVKVTVNGNEVKSEVPGINLDGTTMLPVRAVSEALGAEVKWDQNTYTA
EIEASPTFKSLDEIARHLKNYNIHSVDYHTDGSGHTEFIAIYEGVFFMGNENEYNETVKEIAKQAITLGTTDFIFADMLN
QSVEGTTDNARAFYNGKLTETQFFDTLIREDIEEESELDS
>lcl|NZ_CP026520.1_prot_WP_042230286.1_4710 [locus_tag=PC41400_RS23740] [protein=SRPBCC family protein] [protein_id=WP_042230286.1] [location=complement(5232788..5233213)] [gbkey=CDS]
METGDKVTITVEATVHAPVEKVWEYWTEPQHITKWSFASDDWHAPKAENDLRVGGKFLTRMEAKDGSFGFEFGGEYDEIR
INELISYTLGDGRNVTITFIGQENNTKVIEAFEAEAANSVEMQKAGWQAILDNFKKYSESV
>lcl|NZ_CP026520.1_prot_WP_084706601.1_4711 [locus_tag=PC41400_RS23745] [protein=hypothetical protein] [protein_id=WP_084706601.1] [location=complement(5233367..5234089)] [gbkey=CDS]
MWNSIGMLSLLGCVALFVWAVVAVIRKNGKAKNIFMGSGACFVLMIVGAVNAGPEKPLNVPLSTPAGAKVPSADQTPVVT
PEQGTQTQQGGSSTDENREDDADTDTTVIVPYKTAKPTVTPRPTVTPKPTAKISPAPSVSPTIKPGASNTPKPTATNTSK
PTASPTAKPGTTLSPKPTASPSPKPAASASPKPAATQKPTPKPTSKPETATAKPTPTPATKPSTPSPPSTPKPTPKTSKG
>lcl|NZ_CP026520.1_prot_WP_042230287.1_4712 [locus_tag=PC41400_RS23750] [protein=immunity 51 family protein] [protein_id=WP_042230287.1] [location=complement(5234290..5234640)] [gbkey=CDS]
MDTNNFEETIKPFFWVAHKSSFSVCLNAGSYKPEIFETREDEGFEGNGYDWGSLARVFLDERKPHLAEYVRFDPEAGMFC
AYSSNPESLKEFIVDFKKACEDDAVINDLFSRAELD
>lcl|NZ_CP026520.1_prot_WP_241688725.1_4713 [locus_tag=PC41400_RS23755] [protein=hypothetical protein] [protein_id=WP_241688725.1] [location=complement(5234630..5234941)] [gbkey=CDS]
MAQYVGVDYIKFSISTIESVTDWMKKNDNIAQPLLANIDLLVMISRKYTTDANLLLEKRKIQEWKQVFNDWFARCESKIP
AKYREGIKENGDELFKELEQYGH
>lcl|NZ_CP026520.1_prot_WP_042230291.1_4714 [locus_tag=PC41400_RS23760] [protein=hypothetical protein] [protein_id=WP_042230291.1] [location=complement(5235184..5235777)] [gbkey=CDS]
MDTKKNIENISLLKEKMSDWRKLSDEELFLAVKEFEKMPRLEVSIYYQDLFNDPKFSKTLLDIYNKHKDNTKLVVLLVSA
IGNMIQRYDLPESKEIYEFMLENSDKSKIGPYVAIFLPRLKNFESYDKKWEYFMNIKKMSPKKVAESSFETIMDLYLEKI
PEHYKTEAIQYFNKKVEESNNEYGKQYYQDIINKIKS
>lcl|NZ_CP026520.1_prot_WP_042230293.1_4715 [locus_tag=PC41400_RS23765] [protein=ADP-ribosylglycohydrolase family protein] [protein_id=WP_042230293.1] [location=complement(5236007..5236936)] [gbkey=CDS]
MELSLDRFHGCLAGLAAGDALGTAVEFSRPGTFEPLRDMVGGGVFGLKPGQWTDDTSMALCLAESLLAAGGFDPVDQMNR
YVSWFREGYMSSTGTCFDIGNATREALLLFERTGEPFCGSLDPYSAGNGSIMRLAPVPMFYAADPAAAVGFAVQSSRTTH
GTAECLQACGLLACYLLAGLHGWSKAELLAPDAYREWTAAAGLELSGGLAEILGGSYRRKEPPAIKGSGYVVQSLEAALW
AFDKSSSFEEGALLAVNLGDDADTTGAVYGQIAGAYYGYGGIPEPWREKLAMRGKIGEIARNLHDLQRA
>lcl|NZ_CP026520.1_prot_WP_129112531.1_4716 [locus_tag=PC41400_RS23770] [protein=hypothetical protein] [protein_id=WP_129112531.1] [location=complement(5236900..5237190)] [gbkey=CDS]
MNEHRHADLSPWHCAQLPNALCGAVCWRLPAPELRAMRRKTAGTCNGTGVMCRSTQPACFKAEKGDRPPAEEFLYIQLSW
EAETLWNLAWIDFTAA
>lcl|NZ_CP026520.1_prot_WP_042230295.1_4717 [gene=sdaAA] [locus_tag=PC41400_RS23775] [protein=L-serine ammonia-lyase, iron-sulfur-dependent, subunit alpha] [protein_id=WP_042230295.1] [location=complement(5237436..5238323)] [gbkey=CDS]
MRFRTLKELADICTAENKSISEFMLEEQAKESGRSTEQEFGKMAAYYKIMKEAVQKGLTEDTTSRSGLTGMDAQRVMSYM
SGSVPSVGEEACRAMAYALAVSEVNASMGRVIATPTAGSCGIIPGVFVSAQQRFGWDDEHLVMGLFSAGAIGYIIANNSF
VSGAEGGCQAEVGSAIGMAAGAMVELRGGTPAQAVHAVGLALKNTLGLICDPVGGLVEIPCIVRNGFGAVNALAAADMAL
AGVRSAIPSDEVIQVMYEAGSAMPEKYRETAKGGLAQTPTGKKMMQELKLSRKRT
>lcl|NZ_CP026520.1_prot_WP_042230297.1_4718 [gene=sdaAB] [locus_tag=PC41400_RS23780] [protein=L-serine ammonia-lyase, iron-sulfur-dependent subunit beta] [protein_id=WP_042230297.1] [location=complement(5238320..5239000)] [gbkey=CDS]
MRFKDVFSIIGPSMIGPSSSHTAGAIRLGRVARQLFGQQPDKAELTFYGSFADTYRGHGTDLAIVGGLLDYETDNSRIPD
SLEDAEAAGVEIVFHTSKGLTVHPNTAKIVLYAGGQELSMVGASIGGGNIEILGVNQFDLKFTGEFPTLVVSHHDRPGMI
ADITAILQKHNVNIGYMDLDRKGRDGDAMTVIEADSVITDELIGEILRLSEVDGVKIVDLKERGQS
>lcl|NZ_CP026520.1_prot_WP_042230299.1_4719 [locus_tag=PC41400_RS23785] [protein=dihydrodipicolinate synthetase] [protein_id=WP_042230299.1] [location=complement(5239327..5240397)] [gbkey=CDS]
MAARKNLSPDMLAALHDGVAIPAHPLALDENRQFDEKHQRALTRYYIASGAGGIAVGVHSTQFEIRDEEVGLLEPVLRLA
AEEVERADLRRPFFKIAGICGPTEQAVREAELAVKLGYDIGLLSMGGLTGYDEAQVLERTRAVAEFIPVFGFYLQPSVGG
RIFSFDFWRQFADIPGVVAIKMAPFNRYQTIDVVRAVCCSERRDEIALYTGNDDSIVTDLLTNYRFQVNGQTVEKKIVGG
LLGHWAVWTHKAVELLEAVKKIRDNRDIPAEWLTRNLEVTDANAAFFDPAHHFEGCIPGIHEVLRRQGLLQGRWCLNPSE
ELSLGQMEEIDRVYRDYPHLNDDEFVKANLSEWLAD
>lcl|NZ_CP026520.1_prot_WP_042230302.1_4720 [locus_tag=PC41400_RS23790] [protein=NAD(P)-dependent oxidoreductase] [protein_id=WP_042230302.1] [location=complement(5240400..5241416)] [gbkey=CDS]
MKTVQELEAKLAKPSEALINDMANLEGDIMLLGIGGKMGPSLARLAMNAIKDAGVDKKVYGASRFSSGGLREELEGDGVK
TFAVDLLDDEQLRGLPDVKNIIYMAGNKFGTTGNEHFTWAMNSYLPGRVAEKYRDSRIVVFSTGNVYPITPVHQGGATES
TPPASVGEYGQSSLGRERVFTYFSHKYNIPMAIYRLNYAIDMRYGVLLEIAKSVKAGKPIDITMGHANVIWQGDANEIAI
RCLNHCSTPPAVFNVTGPETLSLRWVANEFGSRLGVEPIFEGRESDTALLSNSSKASRTFGYPKVSVLQMIDWIAEWVEA
DGHTWNKPTHFQEREGKF
>lcl|NZ_CP026520.1_prot_WP_174446815.1_4721 [locus_tag=PC41400_RS23795] [protein=YvaD family protein] [protein_id=WP_174446815.1] [location=complement(5241999..5242394)] [gbkey=CDS]
MKGLKPFFFVTDIGFIVYWLVTFFHLLPDSALFKDYDNPVIVAWNWSFFPLDLLISFTGLLSIYLQRNKRAAWRNWALIS
LVLTFCSGLQAIAFWGFRGDFDAAWWAFNLYLMIYPLFFLRYFIAREPSSA
>lcl|NZ_CP026520.1_prot_WP_042230303.1_4722 [locus_tag=PC41400_RS23800] [protein=amino acid ABC transporter substrate-binding protein] [protein_id=WP_042230303.1] [location=5242497..5243345] [gbkey=CDS]
MKFFRKGGIALTAVFACTLLLGGCGKTATPAPAPSPSPSPGAATPAAGAGTLDAIKSAGKIRIGTEGTYAPFTFHDKSGK
LTGFDVELAEEVAKRLGVQAEFQETNWNGMLAGLDAKRFDLVANEVSIREDRKEKYDFSDPYIISKAVLIVREDNKDITK
LADLKGKKSGQSLTSNLADIAKKNGAEIVAIEGFNQAIELLISKRIDATINDNLSYLDFKKQKPDAPIKVVAETTEASQS
AFLFNKNNNDLAKAVNEALASMKKDGTYLSISNKWFGTDVSK
>lcl|NZ_CP026520.1_prot_WP_042230304.1_4723 [locus_tag=PC41400_RS23805] [protein=amino acid ABC transporter permease] [protein_id=WP_042230304.1] [location=5243377..5244054] [gbkey=CDS]
MSERSERLLHIFMESLWPMLKAGLQYTVTLTLISFALGLVLAFGTALARLSRWSLLAGIAAFYVWIIRGTPLLVQLFIIY
YGLPSLGWTIDAFPAAVLAFTLSVGAYNSEVIRAAILSISRGQWEAAYSIGMTRRQALRRIVLPQAARVSVPPLSNSFIS
LVKDTSLAATITLTELFQIAQQRTAMYYEPMLLYCEAALLYLLFSTVLTYAQRRLERRFDRYAAR
>lcl|NZ_CP026520.1_prot_WP_042230305.1_4724 [locus_tag=PC41400_RS23810] [protein=amino acid ABC transporter ATP-binding protein] [protein_id=WP_042230305.1] [location=5244064..5244810] [gbkey=CDS]
MIELSHLKKQFQSQEVLRGISLNVARGETVVVIGPSGSGKTTLLRCINLLEVPDAGTLRIGDAGLDFTGGRKPSSQDILT
LRRQTGMVFQSYELFPHMTALANVMEGPVTVQGKRAAEARDRAQELLAKVGLGDKAGLYPHQLSGGQQQRVGIARAMAMN
PLVLLFDEPTSALDPELVGDVLEVIRQLAREGMTMVIVTHEMKFARDVADRIVFVDGGVIVEEGSPAQVLDSPSQERTRR
FLHRISGG
>lcl|NZ_CP026520.1_prot_WP_042230683.1_4725 [gene=namA] [locus_tag=PC41400_RS23815] [protein=NADPH dehydrogenase NamA] [protein_id=WP_042230683.1] [location=5244899..5245909] [gbkey=CDS]
MKLFEPFTIGNVTLPNRIVMSPMCMYSCENRDGKVTDWHRTHYASRAVGGTGLLILEASAVTEQGRISPQDLGIWSDGHI
EGLRELVGLIRSHGSKAGIQIAHAGRKAELDGPIAAPSAIPFPGGKTPEEMTAAQISETVRAFADAAKRAKAAGFDVIEI
HGAHGYLINEFLSPLSNHRTDEYGGSRQNRYRFLQEIINAVREQWDGPLFVRVSADEYDPNGNSIQDYAFFASLMKEQGV
DLIDCSSGGVVPAKIDVYPGYQVSLADQIKNTAVIPTGAVGLITGGKQAEDILERGQADLIFIGRELLRDPYWPRTAARE
LGFEPAFAPKQYTRAW
>lcl|NZ_CP026520.1_prot_WP_053228713.1_4726 [locus_tag=PC41400_RS23820] [protein=phosphodiester glycosidase family protein] [protein_id=WP_053228713.1] [location=complement(5246284..5247903)] [gbkey=CDS]
MRRGLDIAGYAFSRLGKRGSLTEGAGKRKAQSDVAGAEVLNEGGGRKGPGTKAMYAGAKGALRMGAAALLIGGILFAAGP
ARQALADDAVTSGSRQINAGGKSFSVKWVRVDVTDPYLEVVPVTASRGIGYDESFSSMIARTGAVAAVNGTFFNAYEKDA
SIRYPNGLMLKDGELLHSGENQSLIVDADKTVDIRKLKMSYTITATHGGKDYTFSPWGTNKDYGDSQPDQVVWFTPEYKA
AVSKSGTTKVVVRDGVITAITGDAVYVPYDGYVCLIGNSSNNKNNLLPHLHVGDTVKTKLEVTENGQRVQGAELWQSATG
VGPKLVTGGKVDIDPARDGFTDPKITKQAAMRSFVGADAQGRLVMGTVNGATMNDLAQIALSMGLTDAMNMDGGASSGLY
ANGSMLTAPGRALSNALVVKRHATPQVQVAVNGQLSAGFKGFIGGETTMVPMRPLLNAMNTEYKWDNAAHSLTLKKGDVE
MVMTVGSYYIRVNGTDYWVPAAPQLLDNYLYVPLRFTAEKIGAEVKWNQSLYRADIVLP
>lcl|NZ_CP026520.1_prot_WP_042230306.1_4727 [locus_tag=PC41400_RS23825] [protein=NAD-dependent protein deacylase] [protein_id=WP_042230306.1] [location=complement(5248046..5248807)] [gbkey=CDS]
MSDLPSFETHTENLRRLIAESERIVFFGGAGTSTESGIPDFRSSSGLYNSGAEYDYPPEVMLSRSFFIREPEKFYAFYRG
EMLHPAARPNPAHLALAELERSGRLTAVITQNIDGLHQLAGSKCVLELHGSVLRNKCLSCGEGYGLEAVMESIETVPRCT
ECGGIIKPDVVLYEEGLDAHVLGEAAAQIAAADLLIVGGTSLTVQPAAGMVGRFRGGRLALINLDPTPYDSHASCIIREP
IGKVLAPFAEKRA
>lcl|NZ_CP026520.1_prot_WP_042230307.1_4728 [locus_tag=PC41400_RS23830] [protein=MFS transporter] [protein_id=WP_042230307.1] [location=5249041..5250366] [gbkey=CDS]
MSQTNPVTPAPLPAKGRKLFLSRPVLSWGLNDVANTIFSSNIVTLFFPFYLTQVIGGDARMDQVASTFITYSNALSGLLI
VLLSPLYGVWMDRTGRKKSLMVPFTLLAIGFTLLMALSANWYSSASFLGLPLAVAGVLFFFVIAKFFFSSSVVFYDAMIS
DLGNSREIPLISGFGVALGYIGTLIGLSVYPLASGQQYEQTFLPSALLYLLFALPLILFYKEAAPAPRSGVQTGFLSGYK
EIAATFKEMKKYRPIFLFMVAYFFFNDALQTAISVMAVYAKTVVGFSTVRFLVLYLVATVSSIFGSFVFGYLTRKYGAKR
AVVYVALLLIVAIGLGTTAVNEPMFWAAGVIYGISMGSMWVTSRTLIVELSPADKRGQFFGLFAFSGKVSSIVGPLLYGT
ITFLLQDQGTLASRVALGSLLVMVVIGLIVHKRIPEQHETV
>lcl|NZ_CP026520.1_prot_WP_042230309.1_4729 [locus_tag=PC41400_RS23835] [protein=nicotinate phosphoribosyltransferase] [protein_id=WP_042230309.1] [location=complement(5250477..5251925)] [gbkey=CDS]
MSSIPNLTLHTDKYQINMMYAHWKNGSHRQKAVFEAFFRKLPFGNGYAVFAGLQRLIDYVHRLKFDEPALRYLSEQEENY
DEGFLEELKAFRFGGSIYAMEEGTLVFANEPLVRVEGTIFETQLIETALLNFINYQTLIATKASRIKNVSPDDTFLEFGT
RRAQEADAAVWGARASYIAGFDATSNVLAGMMFGIPSKGTHAHSWVQGHATEQEAFDTFAEALPNQVTLLVDTYHTLKSG
VPHAIQTAKKMEEQGKRMQAIRLDSGDLAYLSIKARKMLDEAGLEYVKIVASNDLDENTILNLKAQGARIDTWGVGTQLI
TAADQPSLGGVYKLVAIEENGVMVPKIKISGNPEKVSTPGWKDVYRIIGSDTGKAIADYIVLHDETDVQEGRKIKLFDPV
HPYIYKYVDEYKAVKLLKPVFEDGKQVYEVPSLEEMRAHHRKQLDQFWPEYLRKLNPEKYAVDLSTKTWEMKMKMIENFN
FG
>lcl|NZ_CP026520.1_prot_WP_042230310.1_4730 [locus_tag=PC41400_RS23840] [protein=isochorismatase family cysteine hydrolase] [protein_id=WP_042230310.1] [location=complement(5251968..5252519)] [gbkey=CDS]
MKALIVIDYTYDFVEGKLPVGDPAIAIEERITALSEEFLEEGGFVVMAVDFHRLNEPFHPETKLFPPHNIEGTSGRELYG
RLNTLYREKEGQILWLDKTRYSAFCGTDLELQLRARGMEEVHLAGVCTDICVLHTAVDAYNKAFGLIVHEDAVASFNPDG
HTFALNHFRTSLGAQVVRGSEPV
>lcl|NZ_CP026520.1_prot_WP_042230312.1_4731 [locus_tag=PC41400_RS23845] [protein=sporulation protein] [protein_id=WP_042230312.1] [location=complement(5252753..5253526)] [gbkey=CDS]
MTFFQRMLASIGFGAAQVDTRLSKNAYYPGEEVEGVVHIRGGQVEQRIEEIYISLVTEYKREVNNSRVKETREIAKHLVS
DPFTLGPNEEMEVPFRFRLAEHTPLTLGHTPVWFSTGLDIASAIDPSDSDRVEVIPSPEMQVVFDAADRLGFRLREAEVK
QSPRFGGRIGLLQEFEFVPASRFRGELDELEISFLPATEGLELLMQIDRKARGLAGFIAEAVDADETLLRLRIHPADLSK
GSSYMVGLLAELIQDHL
>lcl|NZ_CP026520.1_prot_WP_042230314.1_4732 [locus_tag=PC41400_RS23855] [protein=hypothetical protein] [protein_id=WP_042230314.1] [location=complement(5254085..5254363)] [gbkey=CDS]
MGSNRLAGIRKFITNEKERTLSNIRNGTIGKEQATGSLNTLYQLAADIQDIHYMKEISSTIALLRSASDNWQRQGALISQ
TIDSAVYRPGIK
>lcl|NZ_CP026520.1_prot_WP_042230316.1_4733 [gene=nagZ] [locus_tag=PC41400_RS23860] [protein=beta-N-acetylhexosaminidase] [protein_id=WP_042230316.1] [location=complement(5254604..5256241)] [gbkey=CDS]
MKQRSEMSLRQKIGQMMMCGFSTSNEFDSAMKPNKKIIDLIQKHEIGGVILFRRNLDNPKQVSELNHELQVFAKDSAGGP
LLISIDQEGGMVARIHEGIACMPGSMAIGATRDAQAAYETARISGVELRAMGINLNFAPCLDINNNPMNPVIGVRSFGET
AEIVGELGAAAVKGYQAAGVAPTIKHFPGHGDTHSDSHHALPMIPHDLERLRELEFAPFVKAIQEGAEVLMSAHVIFPAL
EPDGLPSTLSKRVLTDLLRGELGYNGVIVTDCLEMKAIADHYGTAEGAVMAVEAGTDLLLVSHTYDTQVQTIERLVEAVE
SGRISEERINESVDRILALKRGLNITVGNTAGDISSVGSPEHIELVRETYKKSVTLVKDEGQLPLANDKRTFVIWTEVRE
WTQIDEVVEQKETLGAFLAETVHEVTEERIHTNPSEEEVERVLKASQGYDQVIVVTYNASFSPNQVRIVKELAARDSKLV
VVAGRNPFDLKEFPEVKTFLASYENRPHAMNAVAGVLTGQFPAGGKLPVTVTPEYAYGWSASSRS
>lcl|NZ_CP026520.1_prot_WP_084706602.1_4734 [locus_tag=PC41400_RS23865] [protein=response regulator] [protein_id=WP_084706602.1] [location=complement(5256265..5257890)] [gbkey=CDS]
MYHSQNDSKVMKMNLKALLVDDEINILRNLQTVIPWEKHDIEVVALASNGVKALEMVQEHQPDLILSDIRMPLMDGIALL
EQVNKLERDIQVILLTGFQEFEYARSALKYGAKDYILKPINYEELEAVVERVAFDIREKRKAELDEQKRWSRVRHLAYEK
FLFDILLDYTNVTTRHFFEDENSLEETEFTLLLIDLDNYSQLSRSWNEKERKLKNFAVRNVLQDALEPYELLYSVLQTRE
GEWCVLIRHQRETPFNLGTARGWSDALQLAVKQHVKLTVSLGIYPKHMSVLQLADAYKKLQRALQLSPNVEQILLLEEVN
PGRNEWNNSMWELVEEIVSSLKRRDKAQMEQTFEKLNQSLRVISEHSYVRVEQILHFLVLHLIREMREINIMSDEEERGL
WSNLERSVGVKDLLATIHQLIQDCMNAGMNRKPSELLMVNAKEYIAKNLSNDFGVEEIADHLGISCSYFSLLFKQHFGET
FLENLTRQRIELAKSLLRMSDKSITQIGKQVGYAERRYFTRVFYKFTGMTPSEYREQITNA
>lcl|NZ_CP026520.1_prot_WP_241688726.1_4735 [locus_tag=PC41400_RS23870] [protein=sensor histidine kinase] [protein_id=WP_241688726.1] [location=complement(5257860..5259638)] [gbkey=CDS]
MMNLRKKVFLAFMGFIIIPLLSISIFTYMIFQDITQEKYMEQSELTIKSIGRNIASLFKEANYYSDYWMLGDNLQSQLVE
LNENPPSDEQLNNVTNSLRKIFLTYEPLGTFSVYNLYGENYSSGVIPFTPIPFKTLKDHPIYEEVKNLNGFPRWIGPYEN
PELTGDNNLFSQVRTVNYYETPGSGRIGTMFLQFDFKEMDKIFNFYNQKNSHFLLVNEQGVILYDNRKLTNGSNLNKFLS
QKVDLTETYATSRIKFDGIESLVTVYDIGLEKYGTPKWALVSVTPWDYLSGEMSVILKWVGTIMFLFLISALLFNLLFVN
RYIRVIIRIVSSMKQVERGDLDVRVKVQGNDEMTILSRGFNSLVSRIHDLVEEVKQEQEHKNKAELSLLQAQIKPHFLFN
TLESINALAVQNQGKKVSQLVYRLGTILRISFHHKEEIALGLEIDYLKNYMEIQKYRFEELFDFEIHLPRELENHSILKL
TLQPLVENSIQHGFEGITYQGIITIRVEDGGDHIILWVEDNGIGIPEDVLVRFQYRKKNPSLDTNEPRVGLGVLNVADRL
RIHYGDGYGLYICSEPNKGTQIKCIIPKTTAR
>lcl|NZ_CP026520.1_prot_WP_042230319.1_4736 [locus_tag=PC41400_RS23875] [protein=carbohydrate ABC transporter permease] [protein_id=WP_042230319.1] [location=complement(5259659..5260501)] [gbkey=CDS]
MSKLKNGFLHAFLLLYVFVTLYPLFWGFQNSFKRHVDIVASPFSLIKNFTLDNYANAWTSANIGRYFVNSLYISVVASVV
TIVLASMTAYALTRMYFPKMNKIVTGVFLLAIMIPAGSLLIPLYRFILNVTNDLHIPLYDTHLALILPYIAFGLSLSIII
IMAFIKSLPRELEEAGVMDGLTIYGLFWKIVLPLMTPAIVTVFIINFLGHWNEYVLANVMLSKEALRTLPVAMAAMNNGM
AMNYGALLAATMFSIVPVVVIYAILQEKIIEGLAAGSVKG
>lcl|NZ_CP026520.1_prot_WP_042230321.1_4737 [locus_tag=PC41400_RS23880] [protein=sugar ABC transporter permease] [protein_id=WP_042230321.1] [location=complement(5260503..5261390)] [gbkey=CDS]
MNKVYRNPLAYLVFIIPTLFFYLTFYMFPLLTTFRYGFTSWDGLTAPVFNGLDNFKEAFNDKNFWISVKNNLYFILFSVF
IQVPIIVIFSVLISEVRKLRGFYKTTVFLPSILSTAVVGVLWTVVVFDPTVGLLNKLLESIGLDKLIHLWLADESTAMLS
ILLTNAWQWTGFYIVLVLAAIFGISKEVYEAAEIDGATGFQRALLITVPLIRPILAVIVLLSITGAMKAMDIVMVMTKGG
PYGSTEVMASYMYKVGMTTGNYGYANTIAILIFVFTLIITLLSNLLTKKLEDVEN
>lcl|NZ_CP026520.1_prot_WP_042230323.1_4738 [locus_tag=PC41400_RS23885] [protein=extracellular solute-binding protein] [protein_id=WP_042230323.1] [location=complement(5261491..5262822)] [gbkey=CDS]
MKKKLVTIMAVAVGLTTVLSACGKDGGSTGATTTPTGSAAPKEFTVSYRHTQIKDTAKKRKAMFEDALKEVEANIPGLKF
ESEGVDEVVNRDTKLKAEMAAGNPPKIFELFGGTDTKNYAKTNNLLDLTPILDELGLKDKFVDLSEFTVDGKVYGLPIAG
YLEGMFYNKKIFKDNGVEVPKTYEEFLKACETFKAKGITPMALGGNEAWVLNMLPNTLYVRVGGQSVVDGLQPGQSKWTD
EAFVKGHTMFKDLIDKGYFNKDALALKYSQTQDRFANGETAMIFDGGWANQKYSNPDQSKIAADLGFFNFPTVEGGKGND
LINGSYSNGFGFSAKLNENEKKVVKEYIKVFFTEKYQLRQLKEEGFFPSMKLSSTGDKPIIAEMLKASEGKKTFPAFDSM
IQPKVKLDLESGIQAFVGGKKSVQDLLAEVQKSQEAENKKTAK
>lcl|NZ_CP026520.1_prot_WP_053228714.1_4739 [locus_tag=PC41400_RS23890] [protein=sugar ABC transporter permease] [protein_id=WP_053228714.1] [location=5263187..5264131] [gbkey=CDS]
MNTVDESLPIPVGNARSIRTASWWNRVREAGIGYLFLLPSLAGFGIFLFYPFFQSLYLSVHSTNLRGKIAEYVGFENYRE
LLTSEKFYHSLGVTAQFALLTIPTTIAAALLLAALTHRGLKGKRVYQFIFSLPMALSVGTASVIWALLFHPTAGMLNYWL
GLLGQAPVAWLSDPSWALISVSLMTVWMNLGFAYIVLLSGLQSVPEDMYDSAKIDGAGPMRTFFRIVLPLVSPTLFFVGV
VSLIGAFQSFGQIQILTKGGPMDTTNVMVYSLYQEAFVNFRFGSGSAQAILLFVIVLFFTFVQFGVLERKVHYS
>lcl|NZ_CP026520.1_prot_WP_042230324.1_4740 [locus_tag=PC41400_RS23895] [protein=carbohydrate ABC transporter permease] [protein_id=WP_042230324.1] [location=5264128..5264949] [gbkey=CDS]
MIRSLQKTAVYVFLTTAAALVLYPLFFTYASTFMTAGEAGQFPPPLLPESLYLGNLKKVMELVPIGAFLTNSFTVSLLIM
AGQLFTASLAAYAFAFVRFPGKKFVFLLFLSTMMIPWEVTMIPNYLTIRSWGWTDTYAALVVPFLATGFGTFMLRQFFMQ
LPGELFEAAKIDGCGHIRCFFSIAFPLSKPGLATLGVYSFLSAWNSYLWPLLVTNDPAKRTVQIGISMLQFEEFTSWNLV
LAGVALVLLPSLLLLVLGLKQLVRGLTVGAVKG
>lcl|NZ_CP026520.1_prot_WP_042230327.1_4741 [locus_tag=PC41400_RS23900] [protein=ABC transporter substrate-binding protein] [protein_id=WP_042230327.1] [location=5264994..5266376] [gbkey=CDS]
MFSYKPSHRTSVMAMLLASALLMQACGGANETAQSSNKETTGSSAKENAASPVKITFWHSMSGELGTVTDKLISDFNASQ
KNVQVEGIFQGTYDESLNKMKASMDSKSGPSLIQVYEIGSRYMIDSKAITPVQKFIDADKYDLSQLEENITQYYTIGGKL
NSMPFNTSNPILYYNKDMFKAAGLDLEKPPVTFDEITKAAQALSKDGKAGASFAIYGWFMEQFFAKQGAEYVNNGNGRTA
PATESYVNKEAGLKALTWWKSMVDNKTALNLGRKTDDTKKAFAAGQIAMMLDSTASLKGIVSTAEGKFQVGTAPLPKPAD
AAEGGVVVGGASLYITNNKSEAEQKGAWEFIKFLTSPQSQAYWHVNTGYFPITKKAYDEQIVKDNLVKYPQFKTAVDQLH
GSKANTATQGAVMGVFPEARQIVEGAIEEALNGKKTPQAALDEAAKSITEKIGKYNQTVK
>lcl|NZ_CP026520.1_prot_WP_053228715.1_4742 [locus_tag=PC41400_RS23905] [protein=diguanylate cyclase] [protein_id=WP_053228715.1] [location=complement(5266491..5267402)] [gbkey=CDS]
MYFSIPIAPQTIVDFRGYAVLLAVHFGGIYSGFIAAVLIAAARIALFGGPNLSAFIGAANVLFIALAGGLVLRSADKRYW
SRWTKMMAVVLVTTGAAIFSAIGTTSLPIMPVILGTLAVGGFFIGYMVDYIVRAHQLFLLMKEASHVDFLTGLNNHRTFD
RLFNELVQAADRKQEELSLLTLDIDYFKKINDTYGHPAGDAVLQQIGVLLRDSSRRMDEVCRTGGEEFAVLLPDCGRKEA
LHYAEQIRSRVERHSFLLGDGKAIRLTISIGAATFPQVSPDLLIEEADKALYRAKHNGRNQVA
>lcl|NZ_CP026520.1_prot_WP_042230331.1_4743 [gene=lpdA] [locus_tag=PC41400_RS23910] [protein=dihydrolipoyl dehydrogenase] [protein_id=WP_042230331.1] [location=complement(5267726..5269141)] [gbkey=CDS]
MVVGDASLDIDVLVIGAGPGGYVAAIRAAQLGKSVLIVDKSEWGGVCLNRGCIPSKALISAAHQYEAAQHSEVMGVSVEN
VKLDYAKTQEWKASVVKKLTSGVAGLLKGNKVQMFQGEVMFINENEARVFNEQEAPRYRFKHCIIATGSRPIELKPFPFG
GRILSSTEALELKEIPKSMVVIGGGYIGIELGQTFAKFGTKVTVLEGSDTVLPGFEKEMSRLVAKNLSKLNVEVFTEALA
QSSEQTENDVTVTFTVKGEEKKVTADYVLVTVGRRPNTDGELGLDLINVKLTDRGLIEVDEQGRTSIPHIFAIGDIVPGL
ALAHKASYEGKVAAEAIAGQPSVVDYKAIPAVVFSDPEIASVGLNETEAKEKGMNVSVGKFPYAANGRALSLNATEGFVK
IVANKDTGLVVGSQIVGPEASNLIAEIGLAIEMGATLEDIALTIHAHPTLGEIVAEAAEGALGHPIHMLGK
>lcl|NZ_CP026520.1_prot_WP_042230333.1_4744 [locus_tag=PC41400_RS23915] [protein=dihydrolipoamide acetyltransferase family protein] [protein_id=WP_042230333.1] [location=complement(5269146..5270486)] [gbkey=CDS]
MARFEYKFPELGEGIHEGEIVKVNVKAGDKVDDDTILMEVQNDKAVVEVPCPVNGQIQEVLIKEGQVCHVGEVVMIIDAE
GEIPESETSAAAPSPEAAAPAPSNAPSTELPKKEDQGAPTAPAPEEAKKSEGGKPSAAPKDILATPSVRKFAREKGINIA
EVSGTGKNGRITREDIEGFVSGGGTSATAAQADQSQEKAAPSKAASAPAAQGELEEERIPFKGIRKVIANAMVKSKHTAP
HVTLLDEVDVTQLVALRTKAKPLAEKKGVKLTYLPFIVKALVAACRQFPVMNAMIDEEKQEIVYKKYFNVGIATDTDNGL
LVPVIHHADRQNIWMIADSIRDLATRGRDGKLTPHEMKGGTVTITNIGSAGGMFFTPVINYPEVAILGTGRISEKPVVKN
GEIVIAPVMALSLSFDHRLVDGATAQNFMNYIKQLLNDPELLVMEV
>lcl|NZ_CP026520.1_prot_WP_009673352.1_4745 [locus_tag=PC41400_RS23920] [protein=alpha-ketoacid dehydrogenase subunit beta] [protein_id=WP_009673352.1] [location=complement(5270538..5271518)] [gbkey=CDS]
MAQMTMIQAIQDAMRVELARDKNVVLFGEDVGHVGGVFRATEGLQKEFGEDRVFDTPLAESAIGGLAVGMATQGFRPIAE
IQFVGFIYEALDQMAVQAARMRYRSGGSYQAPIVFRTPFGGGVKAAELHTDSLEGLMIQTPGIKVVIPSNPYDAKGLMIA
AIRDNDPVFFMEHLNLYRSFRQEVPEGEYTVELGKANVVREGSDATIITYGAMVHTSLKAAEEIEKTRGVKVEVIDVRTL
MPLDIDTIVESIKKTNRAIVVQEAQKTSGAAAEIIAQINEKAILHLEAPVLRVAAPDTVYPFAQIEDQWLPTPARIIDGL
NQVLDF
>lcl|NZ_CP026520.1_prot_WP_042230335.1_4746 [gene=pdhA] [locus_tag=PC41400_RS23925] [protein=pyruvate dehydrogenase (acetyl-transferring) E1 component subunit alpha] [protein_id=WP_042230335.1] [location=complement(5271659..5272726)] [gbkey=CDS]
MSKPLYEVSSEQVQPLTIIGPDGTVVNEEKMPKLSDDQLKELMRRMVFTRVWDERAINLGRQGRLGFYAPVSGQEASMIG
SEFALNKEDFVCPGYRDMPQIVWHGLPMYQAFLYSRGHQHGGQVPEDVNVLMPQIIIGAQILHATGIGMAFKKKGEKRVA
ITYTGDGGSSEGDFYEGLNFGGAFKLPVIYMVQNNGYAITTPYSKQTAAQSIAHKALAAGIRGVQVDGMDVLAVYAAVSE
AAEIARNGGGPTLIESLTYRFKPHSMADDTTKYRTKDEEGEWAPKDPIVRFRTYLTQKGLWSEEEEAKVKEEAKATVAEQ
IKKAEQTEKMTVAGLIDSMFETTPKHLEEQKAQFS
>lcl|NZ_CP026520.1_prot_WP_042230336.1_4747 [locus_tag=PC41400_RS23930] [protein=alpha/beta hydrolase-fold protein] [protein_id=WP_042230336.1] [location=5272956..5273684] [gbkey=CDS]
MTDSRLYKRTIVKEEISSKHLDSTRSLRIFLPPGYNEVLSYPVIYCQDGEEFFNFGRIATTMTAGILDHGWEPAIIVGVD
VNVKIRTSEYSPDGENHEAYIRFFAEEMLPFVESKYAVRADKSERILAGDSLGGTVSLHLALTYPHLFARVVSLSGAFLE
PTRSRLQSEGDLSWLELYMLIGLDEEEVKTERGTFDFVEANRLSKALLEQKNVRLFYEEKPGKHVWGFWQKELPAILEKF
FG
>lcl|NZ_CP026520.1_prot_WP_042230339.1_4748 [locus_tag=PC41400_RS23935] [protein=lipoate--protein ligase family protein] [protein_id=WP_042230339.1] [location=complement(5273775..5274599)] [gbkey=CDS]
MESLPLLPEKITLIDRSGDLDADLLFPFALEELLIRKVGEGADPIVHLWRHPQGLVLGLRDSRLPHAPQAADRIAKEGRT
VIVRNSGGAAVPLDPGVVNVTLITPKPVGKIDFHDDFETMYRFLGGVLADWNTGVAKGEIAGSYCPGDFDLSVGGRKFCG
IAQRRQARALSVQAFVVVEGTGAEKAAVARSFYDEAGQGAAPDSYPDVEPGRMTSLAEYAGAGLTSPAFCRLTGEWLARQ
TQVRAADYREFAADEDIAAMMRTMRERYGFKAAD
>lcl|NZ_CP026520.1_prot_WP_042230340.1_4749 [locus_tag=PC41400_RS23940] [protein=ATP-binding protein] [protein_id=WP_042230340.1] [location=5274894..5276198] [gbkey=CDS]
MSISFHEVLFNVLIACSPVIVSPFFDESNPLSKKSYKPMMTLVSILAALALATYPYSSGSGIKQSLFMLPLVSSFLHGGL
LPALSVSASLLLLHGFVFPALLPTDAVFLLICACFAGFYIKFHKIGGRFSAFLCPVLLSAGLGFTFMAFQVFFHRKNQLP
LPPDFFSFHAFSIVLHGISMAIIIYLSQFIRESRVNRLESQRAEKLNVLSELAASVAHEIRNPMTVARGFLQILSQTEVN
EEKKKMYTSMAIEEMDRVQQIITSYLAFAKPQAEITEPVSVSGILVNLVQRIQGYAGGSKVQLETHAPPSLWITANAEKL
IQCLVNLSRNGIEAMPEGGRLRIEAYPEPKSVCISIEDTGIGMNAEEIARLGSPFYSTKQKGTGLSVMVAFRIIETFSGK
VQVTSVPGRGTRVLVTLPQAAKPPQLASAQDQFG
>lcl|NZ_CP026520.1_prot_WP_042230341.1_4750 [locus_tag=PC41400_RS23945] [protein=penicillin-binding transpeptidase domain-containing protein] [protein_id=WP_042230341.1] [location=complement(5276186..5278141)] [gbkey=CDS]
MGKELPGTTGKEKADARAGISLAESGIDGGKSINSVQTDPRRKLLLLLGLTGILALLAARLFWIQVLRPHRFSAYEVDLV
KNSVLQRQKSIVLDTGRGDMYDRNLKPLTGKTIRTAIIFPVHPDYTGSAEQLRKVSEALGVTADDWKSFVKQVKEPFLWT
GSLTNSGGGKSGGQLASSESGESVKEPLELSSSQIALLEASGLSYVQVTDYKTRYPDHPPAAQVIGYIGQNPERISGLYA
RELAAGHLTLTSEIGVSGLEKTFESRLRGIGERTLSVFTDAGKRPLGALNARLISPDQSRYPLQLMTTLDSGIQRSIEER
MTRRGITKGAVVVLDAANADIVAMASKPSFHPGEINAIDGDWANLGVKAAAPGSIFKTVVAAAALEKGVVKPDETFDCHG
AWGKYGFTCWKKEGHGKLTFEEGFAQSCNIVFGEVMRRLDSDELETYARNLGLLTPVGWKGSLTGGAAVAQLDGEEGGQL
FGAQTPRDDEGVRLQTAIGQRDVQVTPLQAANLVVTVLHRGKARSPRLVKEVRFRDGLTLDSFPPKFLPGAKEVLSPRAT
SELLREMRLVVRSGTAGRLDAAKWPLAGKTGTAQVQSGPHETVNQWFIGYGPVDAPRYAVAVVIREAAPEAPSEAVPLAR
EIMDMLADQPN
>lcl|NZ_CP026520.1_prot_WP_042230344.1_4751 [locus_tag=PC41400_RS23950] [protein=methyl-accepting chemotaxis protein] [protein_id=WP_042230344.1] [location=complement(5278277..5280424)] [gbkey=CDS]
MNKRLTQTATQLRERFQKKQALSGGAPKPGGPGGFISSSMGLSNPAKSVGMKLFLIFIVSILFFVLTVGITSYQLSKSVV
RGKVSEASRQTMIQASQKLDLLFGGFEDLTMQMMLDADLQRLLETRNEVKDGTYEQLEVTRKITEKVNSYLFSNKSIKSM
TLLKTDGHMINSNGASLNATTEEITKTDWFQAISKTSGQSLWLETKPKGYFSGEPSVAMGRYLNSTSTSTPIAMLLMEIP
TSILEKELGLIQLGDDSTMYISDASNKLIYAKDQELLAKPSPVTISEEQQKTASGDFLSSDKQTQNVYSKFAGTGWFLVG
SMPVGELVKDANVIFNLTLVIALVAAVIAALIGIFVARMIGRPLAALAGLMKQGAQGNLKVRANLKSRDEIGMLGENFDM
MMEQITTLVRQTNESAQEVLNTASEVANASKITATSAKEIAVATDEISGGAAGLASESERGNELTHHIGEKMKQVMSANL
EMGTAASGVRSSGEQGTKYMEELMDKTGQTEVMTRSMVEKVDNLKESTQSIRKILEMLHNLTKQTNILSLNATIEAARAG
TAGRGFMVVADEIRKLADQSKQSIDVVAQITETIQREIDETVQVLHQAYPLFNQQIGAVKEADGIFGQVQNHMSQFIERL
NAVTESISELDQSQAVLSDAMTNVSAVAEESLATSEEVASLSSEQLSISQNLVKLSDRLENLSNSLKDTLSKFTL
>lcl|NZ_CP026520.1_prot_WP_042230346.1_4752 [locus_tag=PC41400_RS23955] [protein=U32 family peptidase] [protein_id=WP_042230346.1] [location=complement(5280640..5281926)] [gbkey=CDS]
MSIAAAVNERARGRRVRLDKPELLAPAGSLEKLKFAVHYGADAVYIGGQKYGLRSNADNFTFEEMREGVEFANRYGAKVF
VATNIYAHNEDIEGIEDYLRELQEAGISAIIAADPIIMDTARRVAPKLEVHLSTQQSVMNWQTVQFWKEEGIERVVLARE
VSMGDIDEIKRHVDIEIEAFIHGAMCSSYSGRCVLSNHFTDRDSNRGGCCQSCRWKYDLYAKGAPLSDPLAPDFPLYGEE
EDAFMMSAKDLCMIEHISDMIESGVDSFKIEGRMKSIHYVATVVNAYRQAIDSYFENPAAYEVKPLWRDEINKAANRPLN
TGFFYEAPGPEDHIFGPEDKAASYDFVGVVMEYDEQTETAVIQQRNYFKPGAEVEFFGPGGTHFKQRVDVIRDEEGAELD
AARHPLQLIRIQTEQPVKPLDMMRKRTR
>lcl|NZ_CP026520.1_prot_WP_042230349.1_4753 [locus_tag=PC41400_RS23960] [protein=peptidase U32 family protein] [protein_id=WP_042230349.1] [location=complement(5281938..5282870)] [gbkey=CDS]
MIPKPELLITAGTVEEAKRFIEAGADAVLIGEAAFGMRLPGDVTIQDLEELVPYVHDKGAKVYAAVNNVFDNEALLEVEK
YLAAISAIGVDGAVYGDPAVLRMVNKAELKLPLHWNAEMTTTNYATANYWASRGAARAVLARELNLEETLEFKLHAKLEV
QVQVHGLTNIYHSRRSLVANYVEHLGKDASSVDRSMDNGLFLIEQERQEERYPIYEDSNGTHVMSSDDICMLEDLHELLK
GNIDSLKIEGLLKSAEYNETVVRAYRQVIDAYCADPDSYKFNDEMLEPIRRLQDPCRELSFGFFYKEQVY
>lcl|NZ_CP026520.1_prot_WP_042230351.1_4754 [gene=mltG] [locus_tag=PC41400_RS23965] [protein=endolytic transglycosylase MltG] [protein_id=WP_042230351.1] [location=complement(5282884..5283963)] [gbkey=CDS]
MHRPKSRIKRLMTGLLVIVLLLAGCAAAAAYYIRGELQPTSASGQEKKVEIVSGDGMKNIASKLEREGLIRNASVFMYYL
KWKNEGGRFQAGEYAMKPGIGTDEIIAMLNNGQTVKANRLTIPEGYTVTQIADKVGQHDGMEASAFVKATEAYKPAVGEE
LLAGIPADTKLKHRLEGYLFPDTYDIPKDATEQKMITLMVDQLDKRLQSLPADWKDKLKASGLTFHQMMTIASLIEREVV
VDEERPLVASVIYNRLKIKQPLQIDATVQYALGKQKERLLESDLKIESPYNTYKNPGLPPGPIASPSLASIKAALYPAES
KYFYYVTKKDGSQGHLFGETYAQHLANIEKSKQTEKNAK
>lcl|NZ_CP026520.1_prot_WP_009676585.1_4755 [locus_tag=PC41400_RS23970] [protein=DUF1292 domain-containing protein] [protein_id=WP_009676585.1] [location=complement(5284048..5284344)] [gbkey=CDS]
MSVKQTNELRSSFGDDIILFDEQDESTVFRILTEILHEDRHYAVLQSEEMKKEGEVSVFRVSRDSEGTLELETIEDDEEW
ETVSELYDELTFPEHDEP
>lcl|NZ_CP026520.1_prot_WP_009676589.1_4756 [locus_tag=PC41400_RS23975] [protein=DUF1292 domain-containing protein] [protein_id=WP_009676589.1] [location=complement(5284341..5284643)] [gbkey=CDS]
MAKEEDFQQEEAEIIYIPDDEGNEEEFEVVMKFEVDGSDKKYMMVVPVEGDEESDEVYAFRYEEDENGDDLKLYTIEDEE
EWNIVEETFNTLIAEEEEEA
>lcl|NZ_CP026520.1_prot_WP_042230354.1_4757 [gene=ruvX] [locus_tag=PC41400_RS23980] [protein=Holliday junction resolvase RuvX] [protein_id=WP_042230354.1] [location=complement(5284656..5285066)] [gbkey=CDS]
MRTMGLDYGDKTIGVAVSDLLGLTAQGIEVIRRASPQHDLDHLRRLIGEYEVTEIVVGLPKNMNGSIGPRGELSQEFAAQ
LTETFNLPVRMWDERLTTVSATRTLLEADVSRKKRKQVVDKIAAVLILQGYMDSKR
>lcl|NZ_CP026520.1_prot_WP_009676594.1_4758 [locus_tag=PC41400_RS23985] [protein=IreB family regulatory phosphoprotein] [protein_id=WP_009676594.1] [location=complement(5285119..5285379)] [gbkey=CDS]
MDFMDKTVKFNVKADEIETSPKEVLLTVYDALQEKGYNPINQIVGYLLSGDPAYIPRHNNARSLIRKRERDELIEELVRS
YLKEHK
>lcl|NZ_CP026520.1_prot_WP_042230356.1_4759 [gene=alaS] [locus_tag=PC41400_RS23990] [protein=alanine--tRNA ligase] [protein_id=WP_042230356.1] [location=complement(5285494..5288121)] [gbkey=CDS]
MKASEIRSKWLEFFASKGHKIEPSSSLVPHNDPSLLWINAGMAPLKPYFDGRVVPENPRIVNAQKCIRTNDIENVGKTRR
HHTFFEMLGNFSIGEYFKEEAITWAWEFLTDPKWIGFDPERLSVTVYPEDTEAFEFWNKKIGLPEARIYKLDENFWDIGE
GPCGPCTEIFYDRGDKYGDLNDPECWPGGENERFLEVWNLVFSQFNHNKDGSYTPLPNKNIDTGAGLERFASILQDVDSN
FDTDLFQPIMEATCRIAGVKYKENEEFDVALKVIADHVRTVAFAVGDGVLPTNEGRGYVIRRLLRRAVRYGKVLGMDKPF
LYQLVPVVGEIMGVYYNEVVEKREFIEKIIRIEEERFHETLSDGLAILAGMVESTRAAGGRAISGPDAFKLYDTYGFPVD
LTEDFANEQGLAVDREGFEAAMEEQRERARSNRSKSGSMKVQGGALSEFTDKSEFVGYTELVVPAKVLAIVHDNEFVDAA
GTGETCQVILDRTPFYAESGGQVADRGVITGEGVRAAVLEVGKAPHGQHVHSVTVEQGILRTGDAITAQVSSELRGDTVK
NHTATHLLHKALKEVLGTHVNQAGSLVEPDRLRFDFSHFGGITPEELQDVERRVNAQIWANTPVDISLRGIDEAKAMGAM
ALFGEKYGDIVRVVRVGDYSLELCGGCHVLSTGEIGLFKIVSESGIGSGIRRIEAVTSRQAYEYVDGQLQLLRDAAGVLK
ANTADVPKRLDSTLQQMKELARENESLRAKLGRIEAGALEGRLKEVAGIPVLAAQVSASDMDSLRGIVDEMKAKLGSAVI
VLGAVAGDKVNLVAAVTPDLIAKGLHAGKLIKETAAKCGGGGGGRPDMAQAGGKDPSKLQEALDGVEALVTGQIS
>lcl|NZ_CP026520.1_prot_WP_042230358.1_4760 [locus_tag=PC41400_RS23995] [protein=DUF1854 domain-containing protein] [protein_id=WP_042230358.1] [location=complement(5288665..5289198)] [gbkey=CDS]
MPNPYESGGFPEGGDPFDILLLEADKCRFRRTEGGLLLLSCDGRDHEEILVFRTFPFSEPDRYISIRTKEGEEIGIIADL
RQLDDASRRELEKELHLRYLLPRVEAIRSIKQKTSGWQWDIVTQFGPERLQLPNLHEHVFRQGEDRLLLVDTGEHRLEVA
SIRKLDPASLKQLQKIF
>lcl|NZ_CP026520.1_prot_WP_042230361.1_4761 [locus_tag=PC41400_RS24000] [protein=ABC transporter ATP-binding protein] [protein_id=WP_042230361.1] [location=complement(5289182..5291431)] [gbkey=CDS]
MALKEPPAAAEELPPELLRSLGAKPAVSARADMDESGRPDDRWLIAGAGEIGVYASGGACLRRFSPEEVRSLRTQDAPQG
GGCLIADTDAGPVVLVRYTGACAKGLEAAVHAANLLLATGDDGTGGAGGDSAGRCALCGEAAPGRDHDCAAGDKTERSSP
GRRRVLTRLLAYTRPYRGRMVMAALFLIIAVLLELAPPLVTKIIVDDVLQPGGVAETLLIWVAALLGARLVLSALQIARG
FIGVKVGAAMMNDIRRETFEALTEQSLSFYDRRQTSQFIGRLNQDTDSMRQFLTEGVIWVASQVLTVAAILSMMLTLSWK
LTLVVLLPTPLLLAGVWILWPRVRRLWYKQWRALNALQTLVGDSLQGIRVVKAFGREKSEQARYGEANREMMTQTVRMEG
LWQAATPLFMLVTGAGGIIVWYYGGLQVLDGGLTLGTLMAFTVYLTMFFGPLQWFTQAISWTNRAVSAAERVFEIMDTPR
EVADTADAVTPGRLEGSVSLDRVTYGYEKHKPVLKDISLEVKPGEMIGFVGHSGAGKSTLIHLICRFYDPDRGRILIDGI
PLKKLSQESLRAQIGVVLQETFLFDGTIAENIAYAKPEASSLEIMRAAKIANAHDFIVAMSDGYDTRVGERGHRLSGGEK
QRIAIARAILHDPRILILDEATASVDTETERLIQEAVSRLVRGRTTFAIAHRLSTLRSADRLVVIDKGRIAETGTHEQLM
QQEGIYWKLVKAQTETALIRERDTSAEPV
>lcl|NZ_CP026520.1_prot_WP_042230363.1_4762 [locus_tag=PC41400_RS24005] [protein=AI-2E family transporter] [protein_id=WP_042230363.1] [location=complement(5291630..5292700)] [gbkey=CDS]
MNSIFKNGWFAGLIYTLLGLLVLFFLLQIRPILGSVFHFLKAVLTPFLIAMIISYVLNPVVNLLSTRKVPRTFAVLLIYA
VFLTSLTVIFMNVTPMFVKQLGELNEHMPEMTIRAQSLVDGMKNSQLLPDTVRSGIHDAIFKLQKNLSLKISDTIHGIGG
TINMLLTAFIVPFLAFYMLKDFQLMEKTVLAIVPKSRRRQVVKVLVDIDTALGNYIRGQFLVCVIVGVLAYIGYWAIGMP
YALLLAALVAVFNIIPYLGPFFGAAPAVIMAATVSVKMLLLVVVINTIVQILEGNVISPQVVGRTLHMHPLLIIFALLVG
GEVAGIPGLILAVPFFAVLKVVLEHLFLYYIHRRTV
>lcl|NZ_CP026520.1_prot_WP_009676575.1_4763 [locus_tag=PC41400_RS24010] [protein=hypothetical protein] [protein_id=WP_009676575.1] [location=complement(5292922..5293119)] [gbkey=CDS]
MMRCPNCSSKDIGKIGSHQFYCWGCFIELAVNGEKMSVYQVEEDGTLSSLDDLFFEEEMPQIHAQ
>lcl|NZ_CP026520.1_prot_WP_042230366.1_4764 [locus_tag=PC41400_RS24015] [protein=PRC-barrel domain-containing protein] [protein_id=WP_042230366.1] [location=complement(5293134..5293661)] [gbkey=CDS]
MRKARDIIGLPVFCVSTGKRAGTVRDVLLDNEWTILGVMLDARHWYSSCRYIPAGEITACGQDAVTIADMKSVRSYAEAE
SLGLIPLNGGKHKLKELPVITADGLQLGTIEDVYLLEQLGTKIIGFELSEGFVSDLMEGRKWLPMPISLTIGEDAVIVPL
NSRLEAEEIFVSKEE
>lcl|NZ_CP026520.1_prot_WP_042230367.1_4765 [locus_tag=PC41400_RS24020] [protein=cysteine desulfurase family protein] [protein_id=WP_042230367.1] [location=complement(5293719..5294861)] [gbkey=CDS]
MGIYLDHAATTPLHPEVLEAMMPYYTDYFGNPSSIHSYGRAAKTALDRSRSAFSAAIGCSPADLLFTSGGTESNNTALYG
VVETFGGKRAHVITTQIEHHAVLHPAERLARMGCDVTFLPVDMTGLIRMEDLEEAIRPETVLISVMYVNNEVGTIQPIEE
IGRLARSRGIAFHVDAVQALGKLPIDLSVLPVDLMSFSAHKINGPKGIGALFASRKLQLAPQVLGGSQERKKRAGTENVA
GVVGFTKSLEIYNNNYKEYNIQTSEYRQHMERIFLEELGPDGFVVNGNPDVSAPHIFNVSFPGSHTETLLMNLDLEGIAA
ASGSACSSGSLELSHVLRAMDLPESVCKSAVRFSFGFGNSIDQIETAARKTATIVKRIRR
>lcl|NZ_CP026520.1_prot_WP_009676580.1_4766 [locus_tag=PC41400_RS24025] [protein=Rrf2 family transcriptional regulator] [protein_id=WP_009676580.1] [location=complement(5294878..5295297)] [gbkey=CDS]
MKISTKGRYGLTIMMELAARFGEGPTSLKSIAEKHQLSEHYLEQLVAPLRNAGLVKSIRGAYGGYILSKNAEEITAGEVI
RVLEGPISPVDFTEEDDPAKRDLWIRIRDSIAEVLDSTTLASLISYQEEGKRDSYMFYI
>lcl|NZ_CP026520.1_prot_WP_042230369.1_4767 [gene=mnmA] [locus_tag=PC41400_RS24030] [protein=tRNA 2-thiouridine(34) synthase MnmA] [protein_id=WP_042230369.1] [location=5295546..5296628] [gbkey=CDS]
MSKQRVILGMSGGVDSSVAALQLKEQGYDVIGVFMKNWDDTDEFGHCTAEEDAEDVRRVCDQIDIPFYTVNFEKEYSEKV
FQYFLDEYKSGRTPNPDVMCNREIKFGEFLQKGLDLGADFIATGHYARLEERDGELVLLRGNDSNKDQTYFLHALNQYQL
SKAMFPIGHMPKPQVREIAAAAGLATAKKKDSTGVCFIGERNFKEFLSQYLPAQPGDIQTLDGEVKGRHDGLMYYTLGQR
QGLGIGGSGTGEPWFVADKDLERNILYVVQGDKHPRLYSKGLSATDVNWISTRRPQGEFTCTAKFRYRQPDQGVTVTLGA
DGAAEVVFHEPQKAITPGQSVVFYDGEICLGGGIIDKVHR
>lcl|NZ_CP026520.1_prot_WP_042230371.1_4768 [locus_tag=PC41400_RS24035] [protein=potassium/proton antiporter] [protein_id=WP_042230371.1] [location=complement(5296912..5298390)] [gbkey=CDS]
MTEKYIFLIAVLLFAGVLMSKFSSRFGLPALVFFMLVGMGLNRFIYFDNAQLTQWFGLMALIVILFDGGMHTKWTHVKEV
IRPSLMLATLGVLLTTAIVGLSAYWILQVTLKEGLLLGAIVGSTDAAAVFAVLGNQNVKRRLTSTLEAESGTNDPMAVFL
TVSLIELIQMPDTSILLLIGSFLWEMGLGLVLGLVIGRLAVWSLNKIDLDSSGLYPVLALAFAIFTYGSTALLHGSGLLA
VYVMALRLGNADFAYRFSITRFHEGFAWMMQILMFMLLGLLAFPDDLISIAWQGVALALILMFVARPVGVFGSMLGTGFQ
YREKLLISWAGLRGAVPIVLATYPLLAGLEHGRMFFNIVFFIVLLSTLIQGSTISVLSKKLGLTEGSAKNDTHTLELLAL
GKANAEIIEIKVTKGGPVNNRLISDLELPEEVLITAIVRKGKIITPHGNTELIEGDLLYTLVSKPMREKVKKIFYEESPA
IKAADLTPENGL
>lcl|NZ_CP026520.1_prot_WP_009674292.1_4769 [locus_tag=PC41400_RS24040] [protein=replication-associated recombination protein A] [protein_id=WP_009674292.1] [location=complement(5298451..5299761)] [gbkey=CDS]
MEMDLFSAAAENDSRGKLLADRMRPRTLDEYIGQEHIVGPGKLLRRAIEADQITSIVLYGPPGCGKTTLAHIISEKTSGD
FVRLNAVDASVKDVREVIEQAKSAKALYGRRTILFLDEVHRFNSARQDALLPAVEQGTIVFIGATTENPFHYVNGALLSR
STLFQLHGLTEKHARIAMERALSDEERGLGFLNIEAEEEALAHIAFMAGGDIRRALNALEIAALTTSADADGVVRITLDI
AEESIRRPTVKADESTQYDVLSAFHKSVRGSSDAALFWFLYAVEKLGMDPMTFIRRLTVACSEDIGLANPQAMVQAVSAM
EAYHKIGWPEAKYIVCQAILFAVESPKSNSIPLAIGRAMSAMDSIGKAEVPLHLRDTHYKGAEQLGHKGYMYPHNYPGHY
VKQAYLPEQLEGEVFFNATEQGMEEKMKINQQRRNR
>lcl|NZ_CP026520.1_prot_WP_042230373.1_4770 [locus_tag=PC41400_RS24045] [protein=Ger(x)C family spore germination protein] [protein_id=WP_042230373.1] [location=complement(5299842..5300945)] [gbkey=CDS]
MRYKLLLFISALLFPLLLGGCNFKDIDKRIFVISMGIDKLDDQKNNLRVSLKLAIPQGNPKEGTEQFDIIVEETKTVADA
VRKAKSKVDKELDFGHMKVFLIGENMAHDDIYQIINWGVRRRDIQLITLLAIAKPTALDVQKVKPESERIPSNALILGLS
KSGTESPYIVTEYLFDYQRRLEERGLDPILPIVKAEGDHFRIDQLALMKKHRIGIELSPNETKLYNLLTTKNLKTNLSGK
MEENLYELNLESSRASYKINARPGEEPVLAYEIHIKATLEENESHENMTRPVLDTIEDATEKEINGQVERFLKRLQKHET
DPIGFGLHYMSRRWDNVSEWEDWQKIYPRLTFKVSTHVRIHSAGTAH
>lcl|NZ_CP026520.1_prot_WP_042230375.1_4771 [locus_tag=PC41400_RS24050] [protein=GerAB/ArcD/ProY family transporter] [protein_id=WP_042230375.1] [location=complement(5300942..5302030)] [gbkey=CDS]
MVTNRYVYYLIAINALISIIIYVPRILLQHRFDGAWTSILLAVVISYMLIRWFTSMLVPFTRQGFPEMMKQHLPKWLNVF
VQLLMIGLWFFAGAITLLSFVDITLRYISPDVPAFLVMIGYLIVVGWACTINPASILFGLEILLVLNCPIVAFFIIRAIT
SSKFDWDAVLQTFTFSLEWPSFQAVAAATFLFTGYINLVVFNRVFEKFHMRGYLLISLSGLLLLLLTMIVPIGMHGSMGV
STYVYPWFSLADSIRVDMFLVERLVFVFYLVYISLFLISTIVHWHVAQSLARGLIRFPAAARGAKFLPLGIVVVFASACM
WMLKFNQEGLIRSGEIFLEIRLVAEFSFVLLIFYMYRKEKKK
>lcl|NZ_CP026520.1_prot_WP_042230379.1_4772 [locus_tag=PC41400_RS24055] [protein=spore germination protein] [protein_id=WP_042230379.1] [location=complement(5302017..5303378)] [gbkey=CDS]
MTGTNATTEQIEQWLAEKLALSVDLENCELTSDGRTMRLIYMKSLCDSKMIRRFFIAPFFEIRSMDLYVQYISSLPSGTK
FTNVVRALDLILSGSVALFVDGKIYLLEALQDVKSSVGAATVEAIVQGPQDAFTESIFTNLNMVRRRYKSSNLKIENSTV
GYLSQTAIALVYDDRKVDMGILTDVKDKFSQMKIDKIQSSGELEKALSPHGFRLVPTMMTTERPDRAIDNLSEGKIIILI
DTSPFALILPSTFFDFFTAMDDKVQLPFVGYFLLVLRYVGLFMTLTLPALYVAFTSYNPEILKMNLTLLIAGSRASVPYP
SFVEVLIMLVMMEFLIEASLRLPKAIGPTATTVGGLILGTASTEAGLVGSIMIILVAAVAISNFVIPINMMSFSVRFAKY
WFVLLAAVFGLLGVVVGVIGSILYLCNLRSFGRPYMRLFHIHEQRGGGTDGHQ
>lcl|NZ_CP026520.1_prot_WP_042230381.1_4773 [locus_tag=PC41400_RS24060] [protein=S-layer homology domain-containing protein] [protein_id=WP_042230381.1] [location=complement(5303693..5305003)] [gbkey=CDS]
MAWKKVLLTLSAAAVLTSAAGQAFAAEHREAERSGGSGAEAPREGQVAGTETGPAAGPLFVDVAPDNRNADAIADLAARG
LIEGTASGRFDPAGALTREQFAKLLASAFGLGTSEGPTPFVDLVSAWSTPYISAAYAAGLVDGVTAVKFMPAQPVTRQAA
ASMVWRWLEARGVEAPSAAPGQSAGDADAWAQASVRSIMALGLRSDSGASYEPKRVMTRGDAAALVSRALALLETTAAER
AAKAAQPLTADKAASLIRFNEQKTAELFLSLFDHELASRTKPPFEAIQPVLSRYYADPGSWRTFYERHLGGIYEAVAIFP
AVSMPVPSFKIVSLTDDRAQVRGKRPEGAYTDSAFLTYTLLKRNGNWVIGGREEEKLTEPFTREEAEGIVARYYKAGSGL
FEREESDRFLFTVTDKGEKLRLSVSKAKGILAMEED
>lcl|NZ_CP026520.1_prot_WP_042230384.1_4774 [locus_tag=PC41400_RS24065] [protein=tRNA threonylcarbamoyladenosine dehydratase] [protein_id=WP_042230384.1] [location=complement(5305050..5305805)] [gbkey=CDS]
MLHQFSRTELAIGPEGLEVMKGSTVAVLGIGGVGSIAAEALARTGVGRIILIDKDVVDITNVNRQIHALTTTVGQPKADL
MQERIKLINPECDAIALRMFYTEETYEQLFDYKPDYVLDASDTISYKIHLIKQCLERKIPMISSMGAANKMDPTKFQVAD
ISKTTMDPIARVVRTKLRKEGIKKGVKVVFSTEEPKKPREDVTQRIVPENAPEIRKAQQPPASNAFVPPVAGLIMVSVCV
QDLLHNAGIDV
>lcl|NZ_CP026520.1_prot_WP_042230386.1_4775 [gene=aspS] [locus_tag=PC41400_RS24070] [protein=aspartate--tRNA ligase] [protein_id=WP_042230386.1] [location=complement(5306012..5307796)] [gbkey=CDS]
MMLKTHHCGQLTKANIGETVILNGWVQRRRDLGGVLFIDLRDRSGLVQIVFNPDFSGEALAVGDRARNEYVLAVKGTVVE
RDAETVNINLQTGEIEVRITEIEIMNAAKNPPFFIEDGIEIDETLRLKHRYLDLRRPEMQQTLMLRAKASRIFREFLDEN
GFLDIETPILTGSTPEGARDYLVPSRVHPGEFFALPQSPQLYKQLLMVSGLERYYQIARCFRDEDLRADRQPEFTQVDIE
TSFLSQDQLLDMMEQLLVRLFRETVGAEVSTPFQRLTYAEAMDKYGSDKPDLRFGLELSTVTDIVADSDVKVFNMVAKNG
GIVKALNAKGCGATWSRKELDDLQPFASRYGGKGLAWITVKEGEWKGPIVKFFKEEEIAALTERLNVEEGDLLVFSADKP
KVVYDVLGNLRLKVGRDLGLIDDKAYKFLWVVDFPMFEYDEDEKRYVAMHHPFTRPKDEDLHLFETDPGQMRAQAYDIVL
NGYEVGGGSMRIYQREVQEKMFAALGFSPEKAKEKFGFLLEAFEYGTPPHGGMAYGFDRLVMLLAGRTNLRETIAFPKTA
SASDPLTDAPSEVDEKQLEQLSIRIALKQPAAKA
>lcl|NZ_CP026520.1_prot_WP_042230389.1_4776 [gene=hisS] [locus_tag=PC41400_RS24075] [protein=histidine--tRNA ligase] [protein_id=WP_042230389.1] [location=complement(5307793..5309082)] [gbkey=CDS]
MSMQKPKGTQDIMPGSVEKWQALESKARDLCSRFRYREIRTPIFEHTELFSRGVGETTDIVEKEMYTFQDKGNRSISLRP
EGTAGVVRAYVENKLYGEPDISKLFYIGPMFRYEQPQAGRYRQFHQFGIEAFGSEDPSLDAEVIAFGYTFYRELGMTGVS
VEINSVGTPEIRGNYKARLQAFLEPFKDKLCKDCQSRFDRNPMRILDCKVDQKYFDGAPELLDDLDEACRTHFEAVKRHL
TAVEIPFTINPRLVRGLDYYTHTAFEYKAAGIGSIDTIGGGGRYNGLVSEVGGQDQPGVGFGLGLERILLVLESQGVQLP
QPPGLDVYLIGLGQAAEDEIVKRLYELRAHGILAEKDYQGRKMKTQMKSADRSAARFVAILGDDELERGEITVKTMATGE
QETIKLENFAETILKRTRSGNENGGVQSL
>lcl|NZ_CP026520.1_prot_WP_009674280.1_4777 [locus_tag=PC41400_RS24080] [protein=helix-turn-helix domain-containing protein] [protein_id=WP_009674280.1] [location=complement(5309574..5309879)] [gbkey=CDS]
MADFHLCPKFENAFELLGKRWTGLIIRVLLSGPKRFKDISDMIPSMSDRMLSERFKELEAAGILIRHVYPETPVRIEYEL
TEKGKGLEPVMDELQKWAETW
>lcl|NZ_CP026520.1_prot_WP_042230391.1_4778 [locus_tag=PC41400_RS24085] [protein=hypothetical protein] [protein_id=WP_042230391.1] [location=5310129..5310914] [gbkey=CDS]
MLPRFFNRIDKLLAIYGAVAAASFAVTLLFAHQSGEKSPPPPAYAYAQITAPNGVELHVAKTKPERIALKAVKGNIMELS
EFGINGGFFWQGNLLSIAVMNDLPVKGAQGEYGSGWFNTGHKRGTLVWDGAARQFTVQQASDAAELKVADRNRYWAQGGI
SMSLQNEADWRKLAKEENMPALNEARMRTGIVYDKSNAVWLVVTPTPCTGEQFRAAIAGSLGRSGLVDGVFLDGNGSSQL
KVAQAALPGDKRQIYQMIALH
>lcl|NZ_CP026520.1_prot_WP_042230393.1_4779 [locus_tag=PC41400_RS24090] [protein=FAD-dependent oxidoreductase] [protein_id=WP_042230393.1] [location=complement(5311086..5313068)] [gbkey=CDS]
MTKSKNNHSKHAASKSWIVWMCLLIVLTVIAAAAALYWQYRHHNKPKPIAQQTLSEVKSVANLEKQYDVIVVGTDPEGVT
AAVSAARNGQKTLLVDGHDREILGGLMTLGWLNSIDMNYVPNPSVLNPHDVLNKGLFSEWYSKIEGDSFSINTAANAFYD
MVRGEKNIDLMLKLKSIEPLLSNQGQQTRVEGITLTKEDGTKQEIRANAVIDATQDGDIAAAAGVPFTYGLEDLGNKNAK
MAVTLVFKLKNVTPDVWSQISKHLKTDDDPMSDANEVSAWGYKELYNYPALNPERVKMRGLNIGRQDGDMVLINALQIFN
IDGLDAKSRAEAFEIAKKEIPNVVAFLKKNYPEFAGVELAGTAPELYVRETRHIQGEYRLNIIDLLENRDQWDRIAFGSY
PVDIQRLAPTDTGMVVTQPEQYAIPFRSIVPKKVDGLLVIGRAASYDTLPHGSARVIPVGMAEGQAAGAAAKLAKDAGLS
FRELAASKENIAKLQDMLNKQGVDLKPFSMKPQPFMTHKQYEGLKAAVSMSLVGGGYKNEFELDKVSNPQRMVNYITRMK
KIIPSAYGGDPSGSLKGIDEEARKKSALTLDQAAYTMTQGLGIAVPKEGAMAELSKRGIVQESSLATIKDKQNLTNGDTF
MLIMDLHKYAESKGAKPAAS
>lcl|NZ_CP026520.1_prot_WP_164977152.1_4780 [locus_tag=PC41400_RS29970] [protein=hypothetical protein] [protein_id=WP_164977152.1] [location=complement(5313485..5313658)] [gbkey=CDS]
MRRSLKEKQMTQPESQLAHSAYSERIRAGAEQSGMEASGEIAPAQGRFKRTAKVPPR
>lcl|NZ_CP026520.1_prot_WP_042230397.1_4781 [gene=dtd] [locus_tag=PC41400_RS24095] [protein=D-aminoacyl-tRNA deacylase] [protein_id=WP_042230397.1] [location=complement(5313794..5314237)] [gbkey=CDS]
MKVVVQRSKEAQVTVNGEIVGRIDRGLMLLVGVGHEDTEEDAKYLADKIAGLRIFEDEQEKMNFSVKDIGGSVLSVSQFT
LYGDCRKGRRPNFMGAARPEQAEALYDKFNGLLREQGLHVETGAFGQMMDVAFTNWGPVTLILESKA
>lcl|NZ_CP026520.1_prot_WP_042230399.1_4782 [locus_tag=PC41400_RS24100] [protein=bifunctional (p)ppGpp synthetase/guanosine-3',5'-bis(diphosphate) 3'-pyrophosphohydrolase] [protein_id=WP_042230399.1] [location=complement(5314265..5316451)] [gbkey=CDS]
MGIEQLLEKAAQYFKETELNRIKEAYEFAEQAHHGQVRKSGEPYILHPLAVADILVDMQMDVTSIIAALLHDVVEDTTVS
LEAVQDKFGRTCAMLVDGLTKLERIKFKTKEEQQNENYRKMFVAMAQDIRVILIKLADRLHNMRTLKYQSEESQRRIADE
TLEIFCPIAHRLGISAIKWEMEDIALRYLNPQQYYRIVNLMQKKRTEREKYIEDVIRNIQDKLREMGIEGDISGRPKHIY
SIHKKMSSRNKQFNEIYDLMALRVIVDNIKDCYAALGIIHTLWKPMPGRFKDYIAMPKPNMYQSLHTTVIGPKGEPLEVQ
IRTWDMHQTAEYGIAAHWAYKEGNGSPEGTFSDKMQWFREILELQKETDNASEFMESMKMDFFTDLVFVFTPKGEVIELP
AGAVPLDFAYRIHTEVGNRTIGAKINGRIVPLDHKLKTGDIVEILTSKHSYGPSQDWVKIAQSSHARSKIKQWFKKEKRE
ENVAKGREAIERELKRLAFDPQALMTDEKLMEAAKKFAFNDIEDMMSAIGFGGITAAQICTKLTEKLRKEKEAEEAQALE
LTNEVKEVKAAPAEKKSRPVSGVRVKGVDNLLIRFARCCNPVPGDDIIGYITRGRGVSVHRTDCTNIPGENTEEGNRVIE
VEWADSVESNFHVEIEITGHDRRGLLNEVLQVVSESKTMISAVSGRSDKNKMAMIHMTILIRNIEHLHAVVEKIKRVEDV
YSVQRIMQ
>lcl|NZ_CP026520.1_prot_WP_042230401.1_4783 [gene=uraA] [locus_tag=PC41400_RS24105] [protein=uracil permease] [protein_id=WP_042230401.1] [location=5316688..5317944] [gbkey=CDS]
MSNRTIGVDERPSLGKTIPLSLQHLFAMFGSTVLVPALFQVNPAIVLLMNGIGTLLYIFLCKGKIPAYLGSSFAFISPVL
GVLATKSYNSALGGFIIVGIVFVLVALLVKLVGTKWIDVVFPPAAMGAIVAVIGLELAPTAAGMAGFLPKDGAPLDGKAL
LVSIVTLLIALLGNVLFRGFMKIIPILVSIVTGYVLAAFVGIVDFAKVSEASWFALPQVYTPQFDMASIVLILPAALVVI
AEHIGHLVVTGNIVGKDLSKDPGLDRSLLGNGISTILSGFVGSTPNTTYGENIGVLALTRVYSTWVIGGAACFAIVLSFS
GKLAALIMTIPTPVMGGVSLLLFGVIGASGIRMLVESKVDYSKATNLILTSIVLIIGLSGAALKIDSFELKGMALATVSA
IVLSLFFKLVEVLKLSND
>lcl|NZ_CP026520.1_prot_WP_042230403.1_4784 [locus_tag=PC41400_RS24110] [protein=adenine phosphoribosyltransferase] [protein_id=WP_042230403.1] [location=complement(5318029..5318544)] [gbkey=CDS]
MNFKEYIRVIEDFPQPGIRFKDITTLLQNGAAYKEAIDEIARFVKDKEIDLIAGPEARGFVVGAPLAYALGVGFVPIRKS
GKLPGKTIEADYSLEYGKDKLAVHDNAIKPGQRVLIADDLLATGGTIKTCVNLIEQLGGEVVGSAFLIELTYLNGREQLP
AGKDVFTLIQY
>lcl|NZ_CP026520.1_prot_WP_042230404.1_4785 [gene=recJ] [locus_tag=PC41400_RS24115] [protein=single-stranded-DNA-specific exonuclease RecJ] [protein_id=WP_042230404.1] [location=complement(5318559..5321015)] [gbkey=CDS]
MLESKARWRLGAGDEAVAGQLCDALGIDPLLARLLVLRGVSSVPEAEQFLSGGSGHFHDPFLMEGMEPAVSRIRSALSGG
EKIRVYGDYDADGVSSTALMAHLLRGLGANFDTYIPHRVKEGYGLNKGAVDLAKEQGVSLIITVDTGVSAVEEVAYIQEL
GMDVIVTDHHEPPGTLPQPLALINPKQETCPYPFKGLAGVGVALKLAQGLLGRLPEELLEIAAIGTVADLMPLHDENRLI
VKLGLERMRKSAYPGLRALMDVSGVDRSAVNATHIGFSLAPRINASGRLEHASTALKLLVTEDEAEAVSVAEELDLLNKE
RQFIVDELTGLAKAAVQEEQDGALKRGETPKNFLVVSGETWNVGVVGIVASRLVDTYYRPTIVLSIDPDTGMAKGSARSI
AGFDMYLALTHCADLLDHYGGHQAAAGMSLRRENIAEFERRLNVLAGEWLSAEDFVPLLAVDAECSLSEVPLECIEELEK
LAPFGVGNPAPKILFSGLRLDEIRLMGRERQHLKLVLTDAAAEIAAGVEAVGFGKGWLGSLMTPQAPVDIVGELSVNEWN
GVRKPQLMLQDLRIPALQVFDWRGKPAARVVEELAALGERLAALPGRGSARPPALVLFGEEDRAPGGRQAGPVWRCMPDG
SLAADNEAARELKFPEMTDIVLCTAPRSLTQLERALAQADGIRRCYAVFADPAPQDGGAMPTRDQFKAVYASVRQQGMWD
LRNPALLQSFSRRSGLSASMISFVLDVFEELGFIEKAGGVVRPVAAPSKRDLTTSRKYQERLSRAEVDRVCVYTSAKELS
DWIAGCRSPQQLSSAQPT
>lcl|NZ_CP026520.1_prot_WP_042230405.1_4786 [locus_tag=PC41400_RS24120] [protein=cation diffusion facilitator family transporter] [protein_id=WP_042230405.1] [location=complement(5321059..5322012)] [gbkey=CDS]
MSDERFQKAEVAAWIGIGGNIGLTVLKGVVGVMAGSRSLLADAVHSASDTAGSFASLRSLKAPGTPVDHGRSYGRDKAES
IAAILIAVIMTIISVEISMNAVKDMVSGVRKAPESYALIAVIASLVCKEVMFRYNYWVGKKLGSRDLVTGTWEHRSGVYS
SIMALAGVGGALIGEKMGWGFMYYLDPAAAVAVTLLIVKNSYGVITDALQKTMDRALHQEDARDMIDAVQGVKGVISVED
LRAREHGHYVIVELRISVNPRISVLEGNEIARSVKYQLMKKFIHLSDVTIHVFPYDSGYPYKHSIDAAANDSAQMLH
>lcl|NZ_CP026520.1_prot_WP_084706604.1_4787 [gene=secF] [locus_tag=PC41400_RS24125] [protein=protein translocase subunit SecF] [protein_id=WP_084706604.1] [location=complement(5322104..5323024)] [gbkey=CDS]
MNYSNKFNFVKHRNVFFAISGILILAGIVTLLLFQMNYGVDFKAGTSIDVSVGQAITKEKAQEIYKGAGFDQTVTIGGAN
GDRVSTRFDEILGKEQVDKIKAAFEAPYKDKVSMEVNTVSPDIAKELAIKAIIGILIASVAISIYVSIRFEWRFALAAII
SILHDAFIVVAMFAIFRFEVDLPFVAAVLTTVGYSINDKIVIFDRIRENLRFAKTKTPEDLEKLVHDSTWQTMARNINTV
LTVLIAAVCLFIFGSESIKLFALAKIIGLTSGAYSSICIASPLWYLLKRNSLGNKRKAAPAAAAKE
>lcl|NZ_CP026520.1_prot_WP_042230407.1_4788 [gene=secD] [locus_tag=PC41400_RS24130] [protein=protein translocase subunit SecD] [protein_id=WP_042230407.1] [location=complement(5323014..5324246)] [gbkey=CDS]
MNVKRISAFFITVIVTLAVIATTSPGLLGKIKLGLDLKGGFEILYVAEPFTAGQKVTPATLKKAAESIAQRADKLGVAEP
EVLAEGENRIRVRLAGVQNEDEVRQKIGKPSSLTFRDPDGNNMLIGTDFVEGGASVGYDQTNQPVVQIKLKDRSKFKDVT
TKMVGKPLSIFLDEEKLSSPMVNTPIDSDVATITGSFKFEEAQKLADTINLGALPLKLTEKYTQVVGASLGQMSLEQTIR
AGIIASVIILVFMILLYRLPGIIAAFTLITYTWLLLVLFDWINATLTLPGIAALVLGIGMAVDANIITYERIKEEIRSGK
SIPSSLKAGSKASFRTIMDANITNIISSLVLYYIGNGAIKGFALTTMLSIVVSILTNVFFSRFLIHLLIKGNVFKKPGYF
GVKEADIREL
>lcl|NZ_CP026520.1_prot_WP_241688919.1_4789 [locus_tag=PC41400_RS30445] [protein=post-transcriptional regulator] [protein_id=WP_241688919.1] [location=complement(5324359..5324619)] [gbkey=CDS]
MEEESEPERLNDRELNEMIEGLCLSKAEEFQMIGYEHVTGEEIWECINDRYKKKGLPPLHQVVNDILSLKPNQFMNYLTL
SMYRRS
>lcl|NZ_CP026520.1_prot_WP_042230412.1_4790 [gene=spoVB] [locus_tag=PC41400_RS24140] [protein=stage V sporulation protein B] [protein_id=WP_042230412.1] [location=5325242..5326813] [gbkey=CDS]
MTKQSFLKGTMILLAAGIVNRILGFIPRITLPRVIGAEGVGLYQMGWPFLIVILTIVTGGIPLAVSKLVAEAEAERNYRR
SARILKLALTLSVSISGIFTAVCILGARWITSHLLTDERVYFTFLSMSPIIPIIAVSSVLRGYFQGKQDMIPTATSQTIE
TLVRIVMVLLCSFALLPYGISWAAAGAMIGVTIGEIAGMLVLLIHYKLQRKKNMPVNKAQIGTSAAHGRMTQLRRLLTLS
LPITASRLVGAGSYLLESILIAHSLAAAGIATALATAQYGALQGMIIPIILLPSALTSSLSVSLVPSLSEAAARKDLKTI
HKRMHQSLRLALVTGAPFAVIMYVLADPLCLLMYGQKDIAVMLKMLAPVAIFIYLQAPLQAALQALDKPGSALINTLIGS
LVKLLLIVWLASKPEFGILGAVMASCVNVVLVTFLHGLSVLRHLKFTLRGSDVAKVSAAMGIASLCCYAVYRSIGSMPAG
ISFLLASAAALTAYLAAVTALKIVNRSDLSRLLTLGRKIIRKP
>lcl|NZ_CP026520.1_prot_WP_042230414.1_4791 [locus_tag=PC41400_RS24145] [protein=DUF421 domain-containing protein] [protein_id=WP_042230414.1] [location=complement(5326879..5327568)] [gbkey=CDS]
MELLTIFLRTVMIYFVVFVMLRIMGKREIGKLSVFDLVISIMIAEIAVFVLEDIRMPMANGMVPMITLVLIQLLMAFITL
KSQRFRTLFEGRPSVLIEHGKLNREEMRKQRYNLDDLMMQLRQHKVMNVADVEFAILEPSGKLSVVEKYHPSRKPEMGPI
ESGRIRYAGLPVALVMDGKVQEDNLQKVGQTRFWLKSELKKKGISDFKEVFFCSVDHRGYLYVDKKDRV
>lcl|NZ_CP026520.1_prot_WP_042230415.1_4792 [locus_tag=PC41400_RS24150] [protein=SLC13 family permease] [protein_id=WP_042230415.1] [location=complement(5327652..5328971)] [gbkey=CDS]
MHEAVQNAGWQAAAAVVVALLAVGLGLKFVRCRAWLPLAGAAVLLAAGIAGPPAVVIEAVPWRALALALTMGIIATVIEE
AGWIQLTAVRFYRLTSANPQKVLAMSMMLAASASALLDALTAMAALLPLLLAFTRAWKIPPSPFVIGMMLACNIGSGLSV
AGTMPARLIGTTGELGLTGVWLVLGPLHAVLLVLAYMMIRTVYVDKLKVGTERLRELQTAASGMNGDKGDIRNPAIALGL
ILLAYICADLTGVDLIWISGAAAALLFQSVIKRRESLKSFAAGFVPLLALTAGWLLIAGAFVQTGVLDVAAEWLMKASSG
DDQRAVLAVLWLSAAGAAFSDGTAWTAFSVPVVQMVSEQLQAVSPDHAGRLWYVLLSGVHLGSNATSFSSFSSLLAFGVA
ASRKNGLSFREYARIGVPFALFSLTLASLYLVGFWFGNN
>lcl|NZ_CP026520.1_prot_WP_042230417.1_4793 [locus_tag=PC41400_RS24155] [protein=TIGR04086 family membrane protein] [protein_id=WP_042230417.1] [location=5329091..5329483] [gbkey=CDS]
MIPINKVSQVRLSSPFLSGLLYAFIMLIAGTIVTSLILLLSRTQESSLSTLAYITHGLSVFVGGWVAGKRAGRKGWYYGG
MLGIFYALIVWIIGFLAVDTAFDLKTLTLAGFAFAAGALGGMIGVNSSSK
>lcl|NZ_CP026520.1_prot_WP_042230419.1_4794 [locus_tag=PC41400_RS24160] [protein=phosphatase PAP2 family protein] [protein_id=WP_042230419.1] [location=5329577..5330422] [gbkey=CDS]
MYPLQSMTHATVYIVISIYVLLYLSLRKNPFFVAGAFVQDFFTKRKFFFHFAAVMAILFFNKVELWIESKMHYTPDFTAK
IHGLEGDFVAGFQHLFYNNTVTMVTSIFYVIVFPCLMIVSILIYTYQKKYRLFYALCYTVIINYVIAIPFYLFFPVQEVW
AFHPNVKLLILEVFPKFEAEYRPLSGLDNCFPSLHTSISVSMALLASRSGNRLWKWFSLLSCVVIIFSIFYLGIHWLTDM
CGGVLLGIFAARLGLSISEERPFFSRILLPQRRLQKQEVSE
>lcl|NZ_CP026520.1_prot_WP_042230421.1_4795 [gene=yajC] [locus_tag=PC41400_RS24165] [protein=preprotein translocase subunit YajC] [protein_id=WP_042230421.1] [location=complement(5330515..5330835)] [gbkey=CDS]
MSNLAAAANASPYGSLVTYIPLILMFLVLYFLLIRPQQKRTKTRNMMLGNLKKNDKVVTIGGLHGTILEITDDIVVLRVN
DATKMTFDRSAVNTVVTSSAVAEEKK
>lcl|NZ_CP026520.1_prot_WP_042230423.1_4796 [gene=tgt] [locus_tag=PC41400_RS24170] [protein=tRNA guanosine(34) transglycosylase Tgt] [protein_id=WP_042230423.1] [location=complement(5330855..5331991)] [gbkey=CDS]
MAAVTYEHIKTCKQTGARLGRVHTPHGVIETPAFMPVGTQATVKTMSPEELKAMDAHIILSNTYHLFLRPGHELVKKAGG
LHKFMNWDRPILTDSGGFQVFSLSDIRKITEEGVQFRSHLNGDKLFISPEKAMEIQNALGSDIMMAFDECPPYPATYEYV
KDSLERTTRWAERCLKSHARPGDQGLFAIVQGGMYADLRKQSAEQLTSMDFPGYAIGGLSVGEPKHLMYEMLECTTPLLP
TGKPRYLMGVGSPDALIEGAIRGVDMFDCVLPTRIARNGTVMTSAGRLVIRNAKHADDLGPLDPECSCYTCTNYSRAYIR
HLIKADETFGIRLTTYHNLHFLLELMRGVRQAIREDRLGDFRDEFFTKYGLHDNDKGF
>lcl|NZ_CP026520.1_prot_WP_042230426.1_4797 [gene=queA] [locus_tag=PC41400_RS24175] [protein=tRNA preQ1(34) S-adenosylmethionine ribosyltransferase-isomerase QueA] [protein_id=WP_042230426.1] [location=complement(5332012..5333133)] [gbkey=CDS]
MDVQDFDYDLPEELIAQTPLLERTQSRLMTLDKQSGAVGHHTFEQLADYLKAGDLLVMNDTRVIPARLFGVKADTGAKVE
LLLLKPLGEDRWEALARPGKRIKTGTELLFGVQPETAADGTPVVPKQAPLSAVVEAEGEMGARTVRFRYEGIFNEILDKL
GEMPLPPYIKEQLPEKERYQTVYARHAGSAAAPTAGLHFTGDFLDKLKSRGVRLAFVTLHVGLGTFRPVSADKVEEHEMH
AEYYELPQETADLINETTAAGGRIVAVGTTSTRTLETAARQSRAASEAQGGSVHTDSGKPYIRACSGWTSIFIYPGVEFQ
LVDAMLTNFHLPKSTLLMLISAMAGRESILNAYREAVERKYRFFSFGDAMFIY
>lcl|NZ_CP026520.1_prot_WP_042230427.1_4798 [locus_tag=PC41400_RS24180] [protein=SpoIID/LytB domain-containing protein] [protein_id=WP_042230427.1] [location=complement(5333143..5335215)] [gbkey=CDS]
MKNSRISWKVVTAALLALVLAAGLFGSAALPLQAAVPKLDTIRVGLFIEHESYRTPEQVVTLSSDKGFKLGVRTPAGVVN
LAAEGGATLKASLDSYGVRVLETANWSEAKAAYDKLQSLPSDRAYIISRSKKGAAVYQVYYGSYAAKETANTAKAKAVKA
TGSAQAVLSGPLHWSAGTYATAAEAEKQAAAVSLAGADADVVLTPDASGQAGYSVWVGGEADEAGLAAVKAALPGVALQP
ADTASAYLIKRVDATTAAAGVPHYFSGAGSQKTWLDGNGAGIKVAEKSNQVYRGSMEISRYNGKFALVNELPFEEYLYSV
VSSEMDGSFPLEALKAQAVAARTFAIKKGMKYGIAHLSDSTTDQAYRGMLREAEIVRQAVDATKGEVLVDKSGIIGDPLY
SSNSGGQTADASEIWGNAVPYLKSVASPDEGAAKGKPVWYRVALLSGETGYVSGTYLKKTGEKSAAGLEIAASTQGGVNV
RILPSTDSASSPAVAQLKDKERVTLLEQVTESNSYSWFRVYDRAQMEKLLAGAGVKLNGPLQSLEVSKRGPSGRATELKA
NGEVVKVKYPNDLRNVLGGLPSTLFDIEEPGRYTIVGADGKPAAQTAGSVQVIGGTQAKPQKAGADLFVMGADNQVRSVA
SGTASFVIRGTGNGHGLGMSQWGARGLAEQGKTYTEILQTYYTGVWTTKE
>lcl|NZ_CP026520.1_prot_WP_042230430.1_4799 [locus_tag=PC41400_RS24185] [protein=S-layer homology domain-containing protein] [protein_id=WP_042230430.1] [location=5335822..5340075] [gbkey=CDS]
MKLKLRKRRLTPIFLTFLLFFLSVLPVPFGINKAMAATNWQAIGNSLTGLNVEVTHDASAPSLVQFNNEVYAAWSEGYNQ
NTIHIKKYNGTKWDYVDGGAGFQADASEAAMDPVMTVYNNELYITWRETSNKGNQIRVKKYNGSNWTFVDGGGSTGLNMN
SPAAAYTPHLVVYKNDLYLIWVENTNRYYQIRVKKFNGSAWTSADDGGDFGLNANPSLNANSPASVVYNGNLYISWDESD
WPKSRLRVKKYDGSTWSFVDGPKGLNNDDSSYAENGSLKVFNNALYAAYSEDNGSGRQLRVKKFDGSSWASADGGGSVGL
NVDASKDVRSPHFDVYDNALYLIWSEESGTNFGEEIRVKKYDGTNWDEASQGIFALNTSRPVAMRPQLSATNDNLIAAWI
EFNGYQYHIRVMKYDGAAWLPAYEEINVGLNGNLIDAYDPDLKEYKDELYAAWVEAGKIKVNKYSNGSVLPVSGLQGLNV
DPTKVAESPKLAVFHNQLYAFWVERDNANVRVLRVKKFVDGSTWINDEDCCRNGLNADPLNQALNPNLLTTDNGMYAVWA
EAGRVGNDIQVKKFDGSWKSLGRAASYTFGDPSLAMFQNELYGSWLELNGNSYQTRVKKFNGQEWVYADNGEVGGTDDVS
SSLIVFNNELYVVWGENSSDRILLKVKRFDGIRWESVENIADSSGGKSLSALKVYGDTLFATWTEWTGRTERVRVKQFNG
TSWYAADDGGANGLNVDASRRAVAPVMEVHNGDLYVSWFEYNGVSNQIRMARYTGAIIPPEPPVPPVNPVIPAPPTGLVA
TPGDGQAFLQWNSVTGATYYQVYQGTSPGSYGSSSAVTVSGATYSTLITGLSNASTYYFAVQAGNADGVSPLSAEVSVIP
QAGHKTELHTLALSQGTLTPAFDFRTLAYTAEVPNIVSSLTVTASVYDPLSTLKVNGAAVLSGQKSSPINLAAGSNIISA
EVTAADGATKIYTINVTREAASSSSGGNSSGSDNSPSTPSGGQTTNDQNASGTVTFENGQKVINVSVDAAKLIEQLEKGG
DKQTLVIPVNQDADKVSVTLNGDALKAMDNFKQAVIAIETPNGSYKLPAAQIDFAALSAQFGQQVKPSDLSIRVDIAKSD
ATKVNFIGSAADKGQFTVMVAPVDFTVSATYKDKTVSLDQFSKFVEREIPIPSGIDPSKITTAVVLNDDGSIRQVPTRFI
TRGGISYAVVKSLTNSTYVLIAHQKTFTDTEEHWAKDIIHDMASRLVVNGVEDAHFAPNAPITRAEFTAIIVRALGLDNK
GKSLVFKDVDTTDWFNSAVAKAQEYGLIDGYEDGSFQPEKTVTREEAIVMMVRAMKLADLEPNVNDTDIKSILATFADGA
SVDAWARQGVSAAVKSKLINGTDIGLLPKRDITRAESTVLIHRMLEKAELIDGVRTK
>lcl|NZ_CP026520.1_prot_WP_042230431.1_4800 [gene=ruvB] [locus_tag=PC41400_RS24190] [protein=Holliday junction branch migration DNA helicase RuvB] [protein_id=WP_042230431.1] [location=complement(5340552..5341556)] [gbkey=CDS]
MEDRIISANLMMEDEIVEFSLRPRYLAEYIGQTQAKENLKIYIEAAKMRKEALDHVLLYGPPGLGKTTLSNIIANELGVN
IRTTSGPAIERPGDLAAILTNLQEGDVLFIDEIHRLHRTVEEVLYPAMEDFALDIIIGKGPSARSVRLDLPPFTLIGATT
RAGLLSAPLRDRFGVISRLEFYTVDELSYIVSRTCDILQVQIVGEASREIGMRSRGTPRIANRLLKRVRDFAQVKGDGII
TVDLARESLRHIQVDDLGLDEIDHKMLRAIITSFRGGPVGLDTIAATIGEESQTIEDVYEPYLLQIGFLQRTPRGRMVTH
QAYTHLGLPIPEQN
>lcl|NZ_CP026520.1_prot_WP_009671933.1_4801 [gene=ruvA] [locus_tag=PC41400_RS24195] [protein=Holliday junction branch migration protein RuvA] [protein_id=WP_009671933.1] [location=complement(5341575..5342201)] [gbkey=CDS]
MIDFLRGKLAARETEYVVLDVNGVGYRVFCANPYAFGLKEDADVTLYTHYHVREDAHLLFGFPTREEQMLFRQLIDVSGI
GPKVALGILAGGRPETVVAAIQQENIAFLTKLPGIGKKTAQRLIFDLKEKLGALPVTERDLAVSFGLVDPSDEFGTQGVW
GEVKQGLLALGYTEAEADRAWSRIQPKVKDTDPADVIMKQALQALFQM
>lcl|NZ_CP026520.1_prot_WP_009671937.1_4802 [gene=ruvC] [locus_tag=PC41400_RS24200] [protein=crossover junction endodeoxyribonuclease RuvC] [protein_id=WP_009671937.1] [location=complement(5342198..5342701)] [gbkey=CDS]
MRILGIDPGIAIVGFGFIDKTGSKLTPVQYGCIQTEAHTDPGLRLKQVYDAACQLIDKYKPDAMAVEKLFFNRNVTTAFT
VGQARGVIILAAVQRGIPVGEYTPLQVKQAVVGYGKAEKIQVQEMVRMFLKLSAIPKPDDVADALAVAICHAHSSTLHDK
IKEVTKR
>lcl|NZ_CP026520.1_prot_WP_042230433.1_4803 [locus_tag=PC41400_RS24205] [protein=organic hydroperoxide resistance protein] [protein_id=WP_042230433.1] [location=complement(5342983..5343405)] [gbkey=CDS]
MEHLYTAVATAQGGRNGKVTSDDGVLNLEVKMPKELGGPGGAATNPEQLFAAGYAACFDSALNLVMRQKQIRSETQVTAH
VTIGKDEEDGGFKLAVELKVSIPGVEMGTANELVAAAHHVCPYSKATRDNIDVFIDVVQA
>lcl|NZ_CP026520.1_prot_WP_042230435.1_4804 [locus_tag=PC41400_RS24210] [protein=BofC C-terminal domain-containing protein] [protein_id=WP_042230435.1] [location=complement(5343604..5344281)] [gbkey=CDS]
MVFNLLKQLKKKIRLKRKRLIFGSYVLLAAIAYLLAAYEFPKPAYREAGAPLGQAAAPEIDQALKAQQEQTLRQIRSEGG
QRLTYIQRSYVCGEEQQELGLKSADEITALNGEHPDWTLSLNSGGSVIFTQHIDDLSPKCKQSAYFGVDESGNLSLFDGL
PANDKVIRTFFQLNIQYLESTLPKETVHQLHEGIRVSDVAEFNSVLSTFSDYAVQESEKAMSPNY
>lcl|NZ_CP026520.1_prot_WP_053228716.1_4805 [locus_tag=PC41400_RS24215] [protein=LysM peptidoglycan-binding domain-containing protein] [protein_id=WP_053228716.1] [location=complement(5344496..5347579)] [gbkey=CDS]
MKIHMVKKGDSLYKIAQKYNVDLDKVIEANPHIANPDQINVGEKVKIPGMPTKPHHHEGQHHDGQHHDGQYHEGHHHDGH
HHEGQHHDGYHHGGQHHDGHHHEGQHHEPHHLVKHIVQQGDTLWKLSKAWGIPLKEMIEVNTQLKNPSVLMTGETVYIPK
KGHKEQEGVQGYQPHHGGTAHGKKNTAPIQPVVPVPEVVEHKPEVVQPPPSPEINQITKITEINLNLAKVTDHKEEVKIE
LPAQPAPKVELPVIEQPPVQPVPQPPVPEYQLPQLVLPQIELPSYTPQPPPQHPCEPVVPTPYPAYVKEAEAIPDCGGYP
EILPMIQPCPPYAPFVSPYGHHHEMWKKEPCGCDGGLPVPYGHHGAYEGGWGHPQTLPYPSVEIPQGIPSYQAMAAPEEE
FIQPAGHIQAQPYPSQPWFPSQPGLYGGMGVGPDLNTPYTDGAYCAEKPAFGQHPGYPEPYMPFGHEGFEHGGYGNVGGW
PQSHEQAGGWPQGLMPQQWAEQGQHAPGGYGNVGGWPQSHEQAGGWPQGSMPQQWAEQGQHAPGGYGDVGGWPQSHEQAG
GWPQGLAPQQRSEQGQYARGGHEQEGVWTQGYASPQEAEQAAYALGSREQAGVWSQGYASPQEAEQAAYALGGQEQAGVW
PQGYASPQEAEQAAYASGSREQGGVWPQGYASPQEVEQAAYAAGSREQADGVWPQGYASPQEAEQAAYASGIREQGGGWP
QGYASPQEAEQAAYASVSREQAGGWPQDFAPQQQPVQGQYAPGGYEQGGGWPQGFAPQQQPVQGQYAPGGYGEGGGWPQG
FAPQQQPIQGQYAPGGYGEGGGWPQGFAPQPYGSQAFAAGMASIPPSGGKKDCGCGCGGAKKGRGQTPETTNFPAFPSGF
VPYAGGAQQSPYTPFTSYGQPDWERIYPPAFPVSYGYGPDQEGAVNPYAQIPQPAYAVSGPQFQGASNGIEGFQGSEQGY
GDSEARIHLADQEEESDSFLKTQTKAKKTSAPKKSKAPKSSVRTIIRKSGRSAPKQTVSANVPWMNV
>lcl|NZ_CP026520.1_prot_WP_042230437.1_4806 [gene=ilvE] [locus_tag=PC41400_RS24220] [protein=branched-chain-amino-acid transaminase] [protein_id=WP_042230437.1] [location=complement(5347875..5348756)] [gbkey=CDS]
MAQQWIYLDGSFVTKEEAKVSVYDHGFLYGDGIFEGIRIYGGNIFKCKEHLDRLYDSAKSIMLNIPLSFQEMQDALVETI
RRNELRDGYIRLVVSRGPGDLGLDPRRSPKASVIIIVEQLSIYPEEAYINGLKTVSVSSRRNIPDAVNPKIKSLNYLNNI
LVKIQSNLAGVGEAIMLNSQGYVAEGSSDNIFIVKRGVVFTPPCYIGALEGITRAAIIEICERKGYTLKEEPFTLHDVYV
ADEVFFTGTAAEVIAVREVDGRTIGEGKAGPITTNLLQEFRSIVTVDGVKVYE
>lcl|NZ_CP026520.1_prot_WP_042230439.1_4807 [locus_tag=PC41400_RS24225] [protein=prephenate dehydratase domain-containing protein] [protein_id=WP_042230439.1] [location=complement(5348766..5349704)] [gbkey=CDS]
MAKKRIAILGPGTFSEESVHHFLGRDRYEYVPFKLISDVFLATASGETDYSVIPIENTIEGSVTLHIDWLVHEVDLPMQV
EWVYPIDMNLIGLPGSAHRAADDKNGLARLENGELASSPPVSWSNITKVLSHSVALAQCRKFLRTYMPQAEIEQTGSNGE
SVRLVSELGDSNVVAIGTTNAADMYGMELLASKIQDNPDNSTRFILIGREKLDLPASGHTKTTILVTPPEDYPGALHQVL
SAFAWRRINLSRLESRPTKRKLGKYYFYIDVEGSLESVLVRAAFEEIEAIGCQVRILGCYPTYQYETQTPEV
>lcl|NZ_CP026520.1_prot_WP_042230441.1_4808 [gene=thrB] [locus_tag=PC41400_RS24230] [protein=homoserine kinase] [protein_id=WP_042230441.1] [location=complement(5349707..5350669)] [gbkey=CDS]
MAPAAVRVRVPASTANLGPGFDALGMALDLYARIEMSEADRTQIELFGSNMQGIPTDKTNLIYKVAGTLYHHAGLPMPEL
SIRMDSDIPLTRGLGSSASAIVGALAAANALSGADLPPDELFQIASGMEKHPDNVGASLFGGIIAAYWDGHEAKHVRIEP
HERLEALVAVPAFQLSTEHARRVLPDQVPMKDAVFNIGHSSLLVAALAAGELGMIRHAMKDALHQPYRAALVPGMEKILR
EAADHGALGAALSGAGPTLIAFADKSEGRQEELEAFLRETFRAEGIEAETRWLRPSREGVRVWKLAGGEPSFEEFAGGAE
>lcl|NZ_CP026520.1_prot_WP_042230443.1_4809 [gene=thrC] [locus_tag=PC41400_RS24235] [protein=threonine synthase] [protein_id=WP_042230443.1] [location=complement(5350669..5351739)] [gbkey=CDS]
MRYMGLLHQYKSYLPITEKTPLLTLHEGNTPLIRAEKLSEELGLDLYVKYEGLNPTGSFKDRGMVMAVAKAMEEGSSTIM
CASTGNTSAAAAAYAARGGLNCIVLIPNNNIALGKLAQAIIYGAKVIAIEGNFDRALEIVREITAKQPITLVNSVNPYRI
EGQKTAAFEVCDQLGKAPDYLAIPVGNAGNITAYWKGFKEYQAAGKSSSLPQMIGFEAEGAMAIVKGEPILNPETVATAI
RIGNPASWDGAVNAANESGGQINYVTDDEILAAYRKLASQEGIFAEPASAASLAGVIKLKREGHFRQGQSVVCVLTGNGL
KDPHNAIQSVKAEPLVVEDSEAAVMDAIRQLEQVGS
>lcl|NZ_CP026520.1_prot_WP_042230445.1_4810 [locus_tag=PC41400_RS24240] [protein=homoserine dehydrogenase] [protein_id=WP_042230445.1] [location=complement(5351774..5353063)] [gbkey=CDS]
MKKPIKIGLLGLGTVGTGVVRIVEGHREDLEKQTGSAIVVEKVLVQDLSKLRSVSVDRDKLTDQPWDIIRNPEIDVVVEV
MGGVQETKAYILEALEQGKHIVTANKDLMAMHGPEILAKAAEKGCDVFYEASVAGGIPIIRTLVEGFSSDRITKIMGIVN
GTTNYILTKMSQEGAAYADVLKEAQMLGYAEADPTSDVEGLDAARKMTILSTLGFHSEVALADVKVKGISKVSKEDIQYG
KKLGYEVKLLGIAEKHDEHISVSVEPTMVKTSHPLASVNGVFNAVYVYGEAVGETMFYGPGAGEMPTATSVVADLVAVVR
NLNLGVNGQHSTAPYLEKKLKTDEQISSKYFVLLHVEDKAGVLAQITQIFARHEVSLESVFQHPNEHNPKAEIIIITHDA
SQAGMNKVLAEFESLDVIQTIQSVYRVEG
>lcl|NZ_CP026520.1_prot_WP_009675461.1_4811 [locus_tag=PC41400_RS24245] [protein=ACT domain-containing protein] [protein_id=WP_009675461.1] [location=complement(5353166..5353639)] [gbkey=CDS]
MEPKQAAGWSESGFYLVNEQILPEGILKTVQAKELLARGEVKTIHEAVEQVGLSRSAFYKYKDGVFPLNKLDKERIVTIS
MNLEHRSGILSRVLALIAGQEGNVLTINQSIPLQGMANVVISVETSLMGDKAKELVQLLESQDGVKKAAVIGQGSSI
>lcl|NZ_CP026520.1_prot_WP_042230447.1_4812 [locus_tag=PC41400_RS24250] [protein=NUDIX hydrolase] [protein_id=WP_042230447.1] [location=5354033..5354539] [gbkey=CDS]
MITLRLISAALLFNARGELLMMKRSMKRTLSPGMWAAVGGHLESFEMARPRAACLREIEEETGIREDEIPGLRLQYILIR
LKDQDIRQQFIYVGHTDVSPRIRTSEGDLHWIPRTEILNRELPFIFRYALEHYLSNGDTTHTWIGTAVPGSSGIPEVSWT
ALQDPEKT
>lcl|NZ_CP026520.1_prot_WP_042230450.1_4813 [locus_tag=PC41400_RS24255] [protein=aldehyde dehydrogenase] [protein_id=WP_042230450.1] [location=5354569..5355939] [gbkey=CDS]
MNRIHALLDKQRNFFATGATKPLNYRTEQLRRLKAMIREHEEAIMEALRLDLGKSDFEAFATEIGILLEEINHVLKYLKR
WSKPRRVKTPLTHIGSQSRIYAEPYGIALLISPWNYPFQLAISPLIGAMSAGNCAVLKPSELASHTSALLARLLGQTFDE
AYVAVIEGGVETSNALLAEKFDTIFFTGSVGVGRIVMEAAAKHLTPVTLELGGKSPCIVHADANIKHAAKRIVWGKTLNA
GQTCIAPDYLYVHESVKDELLKRMKETVDAFYGDPLADGSPYPNIINDRHYNRLKSLMEAGQLVYGGRTDERSLKIEPAF
ISEVGWEDPVMQEEIFGPLLPVLTYRNLEDVIREVNARPKPLALYAFSESEEIQQRIIHSISFGGGCINDTILHIATPYL
PFGGVGNSGMGGYHGKASFDAFTHRKSVMKQTSRFDLPFRYPNAKNALKRLRSVYK
>lcl|NZ_CP026520.1_prot_WP_042230451.1_4814 [gene=obgE] [locus_tag=PC41400_RS24260] [protein=GTPase ObgE] [protein_id=WP_042230451.1] [location=complement(5356039..5357346)] [gbkey=CDS]
MFVDKAKIYVKGGDGGDGLVAFRREKYVPEGGPGGGDGGDGGDVIFRVDEGLRTLVDFRYQKHFKAKRGEKGRNKCQHGA
NADDMVVRVPPGTLVIDDDTGEIVADLTRHQQEVVVAKGGRGGRGNMRFATAKNTAPEIAENGEEGQERWVVLELKVMAD
VGLVGFPSVGKSTLLSVVSAATPKIGAYHFTTITPNLGVVDLGEGRSFVMADLPGLIEGASQGIGLGHEFLRHVERTRII
VHVVDMAGTEGRDPYEDWKKINEEIKLYNAKLEERPQVIVANKMDIPEAEENLAAFRKQMEEDGKSIEIFPISAVTKGGI
QPLLYRVADMLDAIPDAPLLEEVQEVEERKVYRMEKKNEYDNDFVIRRENDVFVVESPSIEKLMKRTNFTTEEGVLRFAR
ILRNMGIDKALRERGAVDGQTIRLNDYEFEFVEKS
>lcl|NZ_CP026520.1_prot_WP_053228717.1_4815 [locus_tag=PC41400_RS24265] [protein=Spo0B domain-containing protein] [protein_id=WP_053228717.1] [location=complement(5357352..5358125)] [gbkey=CDS]
MGNRLWIPAALLGLLGLIMLAAAPLTVGLRLAAGIVTAACFAAVLYIRHRMKLKEAEEERARIQRLAQQEMLLTINHLRH
DWMNDIQVLLGYIQLKKYDHLHGVVAGVMGKLQHESFMAKLGDPGLIVFLLTAKAEGRHMKLEVELDQEVNIRKLALDGD
AVCKLIPELIEGFKQASAGENEEGGMLSMQLVPEEGSLLLDFVYQGPYNRGLLEEALSLALESGTGRTLAAVVEQNDYQE
QEAALALRLPYRTSVEV
>lcl|NZ_CP026520.1_prot_WP_009675465.1_4816 [gene=rpmA] [locus_tag=PC41400_RS24270] [protein=50S ribosomal protein L27] [protein_id=WP_009675465.1] [location=complement(5358294..5358605)] [gbkey=CDS]
MLKLDLQLFASKKGVGSTKNGRDSIAKRLGVKRADGQTVTAGSILVRQRGTKIHPGNNVGIGSDDTLFAKVEGVVKFERW
GRERKKVSVYPVETAPVAATVEA
>lcl|NZ_CP026520.1_prot_WP_009675459.1_4817 [locus_tag=PC41400_RS24275] [protein=ribosomal-processing cysteine protease Prp] [protein_id=WP_009675459.1] [location=complement(5358610..5358957)] [gbkey=CDS]
MIHVTVSRRSEQDPTIEAYQVEGHAEFDVPGKDLVCAAVSAITVGTVNSIEALTGVVPGTEMEKGWLDVILPLDSEPAQM
AQVQLIIESMIVMLHTIASSYSDYIAIHTAYNEGG
>lcl|NZ_CP026520.1_prot_WP_009675460.1_4818 [gene=rplU] [locus_tag=PC41400_RS24280] [protein=50S ribosomal protein L21] [protein_id=WP_009675460.1] [location=complement(5358969..5359280)] [gbkey=CDS]
MYAIIETGGKQYKVQQGDVLYIEKLEAGAGDVVTFDRVLVVSGDNGLQVGAPVVEGAKVSGKVEKQGKGRKIIVYKYKAK
KNYRRKQGHRQPYTKVVIEKIEA
>lcl|NZ_CP026520.1_prot_WP_042230452.1_4819 [locus_tag=PC41400_RS24285] [protein=Rne/Rng family ribonuclease] [protein_id=WP_042230452.1] [location=complement(5359798..5360961)] [gbkey=CDS]
MKQLIVSCEAASLQAALLEDRKLVEHEAQHPSETQRAGSIYIGRVVNVLPGMQAAFVDIGLDKNAFLYIDDVLPVNMQKQ
PKTKPPIGDLLQPGQRVLVQVVKEPEGSKGARVTTHYSIPGRWLVYMPGAGYTAVSRKIEAEEEKVRLKQVADNLRCGEE
GLILRTVSEGRSEEELRRDLLDLRELWTVIEQRAEAGAAPACIYQDLELLPRLARDVITEDVLEIWVDDEAVHAELQGLL
RQRSPELAARVRLDGGDTPLFERFGIAEELSRSQRRKMWLPSGGYLVIDPTEALTVIDVNTGRFTGTSSLEQTVFRTNME
AAAEIPRLLRLRNLRGIIIVDFIDMEQSAHRDRVLECMAEAVKKDRSKTILLGWTKLGLLEITRKRK
>lcl|NZ_CP026520.1_prot_WP_042230695.1_4820 [locus_tag=PC41400_RS24290] [protein=M50 family metallopeptidase] [protein_id=WP_042230695.1] [location=complement(5361104..5361970)] [gbkey=CDS]
MLQVRFRFHPLFTMVMLLSLATGHFLESATLFGIVFLHELGHFGAAAAFGWRVREVRLLPFGGVLVVDELASVPLREEMI
VALAGPLQHLWLIALGFLCRFLEAGDPEWWSYFIEINAVIALFNLLPILPLDGGKLLQGLLSMGLPYQRTLVCTVWVSIV
LGLMMTVIAIWPGGTGQIHMNVLIVGLFLLYQNWVSRRHLPFYFLRFLMNREANTERLMQKGTQARPIAVARQRKVGEVV
KLFMREKYHPVYVMGSRGTIEGILPEQQVVDTYLTHKNRSCTVSELLP
>lcl|NZ_CP026520.1_prot_WP_042230453.1_4821 [locus_tag=PC41400_RS24295] [protein=M23 family metallopeptidase] [protein_id=WP_042230453.1] [location=complement(5361972..5362907)] [gbkey=CDS]
MENKHNVRQRRLEKIRKLQESPSGKTPRHRQSSELLPFGNPSSEVEPAPDILPLSRRDQEFDDRWNDPEYVWKQKMQREL
QLPAAYYARRSGATDPGDGDGGGYHPRIPYRIRFLRQLVASAMLFGLIWGMFQLQHPLIERGKGIVTTALREPLDFQQLA
AWYQKTFAGAPSFIPAFHSDEENPAVKADAQKRSYFVPLKGKIHETYTADKPFVSVAAEPHTPVYALDAGQVTFAGPKDE
TGYTIIIRHPGGLQSEYGQIEQSRVEVNDWIKGGEAIGQVSRDSKSDKGLLYFAVQKESRYVDPAGVIPFE
>lcl|NZ_CP026520.1_prot_WP_042230456.1_4822 [locus_tag=PC41400_RS24300] [protein=FtsW/RodA/SpoVE family cell cycle protein] [protein_id=WP_042230456.1] [location=complement(5363004..5364158)] [gbkey=CDS]
MLAKLKKLDLTIIFLLMVFMAISSLLVYSADIDDPTISVSWSKLIISYLIALIALMGASFLDYRFLVRFAVYPYAISVIL
LVVVLKFNQKINGASGWFQLPGGFSFQPAELIKVALIIALAAWLHKRRGEDLRLLQDMVPLGLITLLPFGLVFIQPDLGN
AIILIGILIGMVWIGNIKYSHALFALIVFGGGLFAFFYFYSMYPKEISGYLVEHFHFKDHWFQRINTFLDPMQASRDERY
QMENSLRAIGSGGLTGEGYLNGSSIHSNFIPYAYSDSIFVVVGEEFGFRGAAVLLMLYFFLIYRMILIAIQSRNRSGSYL
VVGVVAMYVFQIFENIGMLIGLMPLTGITLPFISYGGSSLLINMLCIGMVLSVKIHDIAIEDDK
>lcl|NZ_CP026520.1_prot_WP_009672982.1_4823 [gene=minD] [locus_tag=PC41400_RS24305] [protein=septum site-determining protein MinD] [protein_id=WP_009672982.1] [location=complement(5364166..5364960)] [gbkey=CDS]
MGEAIVITSGKGGVGKTTTSANVGTALALQGKKVCMVDTDIGLRNLDVVMGLENRIIYDLVDVVEGRCRLPQALIKDKRF
EELYLLPAAQTKDKHAVSPESVKDIVLELKKDFDFVIIDCPAGIEQGFRNAVAGADKAIVVTTPENAAVRDADRIIGLLE
NEKIHAPKLIINRIRQNMVKKGEMLDIDEICSVLAIDLLGIVPDDEHVIKAANSGEPTVMNPTSRAAIAYRNVARRILGE
SVPLMPLDEKPGMLAKLKKFFKLG
>lcl|NZ_CP026520.1_prot_WP_241688727.1_4824 [locus_tag=PC41400_RS24310] [protein=septum site-determining protein MinC] [protein_id=WP_241688727.1] [location=complement(5365024..5365692)] [gbkey=CDS]
MGWEVGTFMATAGKHHVIIKGVKDGLNFLLDDSCELATVLEELRHKVESTHDRILTGPIVRVQVRLGKRSVSDEEKEEIR
AIIGQKGNLLIQSIESDEPEPLVGTAAPITMLKGMVRSGQTLSHEGNLLYMGDVNPGGTILCTGSIYIMGSLRGMAHAGV
DGDEAAIVAASHMRPTQLRIAGIISRPPDEWGFQEAFMEYAYIKDGRMEIDKLNGLHRMVLA
>lcl|NZ_CP026520.1_prot_WP_042230461.1_4825 [gene=mreD] [locus_tag=PC41400_RS24315] [protein=rod shape-determining protein MreD] [protein_id=WP_042230461.1] [location=complement(5365747..5366274)] [gbkey=CDS]
MVRNSLWLILFSLLILEGSLIHWLIPDSWQQSVRISSNFMLIIILYIGLYLNRHSALLYGLGFGLLHDFVYFGHILGVYS
FGFGLVGYVAGLLPRRQTNLIFTTLMTMGVGVLIFEIIKFGINRLFNIVTVNFQFAFTHIMLPSVLFNMLFALAVYIPMR
KLMEALQGQGSRAKH
>lcl|NZ_CP026520.1_prot_WP_042230697.1_4826 [gene=mreC] [locus_tag=PC41400_RS24320] [protein=rod shape-determining protein MreC] [protein_id=WP_042230697.1] [location=complement(5366277..5367155)] [gbkey=CDS]
MKFMGNKRLMVLLVGLICFFILMGLTLSGRTKFYWPEKFLKDSIAWTQSVFYRPAAGVAGFFEDVGKLKTIYDENQILKK
RLTDYALATQQLNALEDQNKRYKEALEFTERQKQVSNYKYRIADVIAASPDPYNGTITVNLGEKDGIKPNMAVINVDGLL
GRVLTVTEFTSNVQLLTDLSDTNNKAPAISATVKGKEYKSAEGQGSYGIIESYDKEKGLLVMNKVDQNDKLAVGDTVITS
DIGKVFPSGIPVGTVVSREPGDFGITQKALIKPFADFRHIREVFIIEIPEVK
>lcl|NZ_CP026520.1_prot_WP_042230463.1_4827 [locus_tag=PC41400_RS24325] [protein=rod shape-determining protein] [protein_id=WP_042230463.1] [location=complement(5367189..5368220)] [gbkey=CDS]
MFGGFTKDLGIDLGTANTLVYLKGKGIIVREPSVVAIRTDTKTIEAVGDNAKKMIGRTPGNIRAIRPMKDGVIADFETTA
TMIRYFLRQAQKDRFMFQRHPNVMVCVPSGITAVEKRAVEDATKQAGARDAYTIEEPFAAAIGADLPVWEPTGSMVVDIG
GGTTEVAVISLGGIVTSRSIRIAGDEMDEAITQYIKRTYNLMIGERTAETLKMEVGSALPMEQPEFLEIRGRDLVTGLPK
TISVTSDEITDALTDTVNAIIDAVKVTLEKCPPELAADIMDRGIVLTGGGALLRNLDKLLAGETGMPVIVADNPLDCVAI
GTGRALDHIHLFKGAKSGAKYRR
>lcl|NZ_CP026520.1_prot_WP_042230465.1_4828 [gene=radC] [locus_tag=PC41400_RS24330] [protein=DNA repair protein RadC] [protein_id=WP_042230465.1] [location=complement(5368370..5369059)] [gbkey=CDS]
MISSIVTLREIPPEERPRERMMQLGASSLSSAELLAILVRTGTYSESALALAQRILSDTGGLRGLVEASREELVRIKGIG
DAKALQIMAGIELGRRISRTRTEEITTVRSPFDVFDLLKDELRYLQKEHFVCLFLNTKNHIIGRETLSIGSLNASIVHPR
EVFRAAIQRSSASIVCVHNHPSGDPTPSPQDIELTHRLTEAGDIIGIEVLDHIVIGDDRYVSLKEHGHM
>lcl|NZ_CP026520.1_prot_WP_042230469.1_4829 [locus_tag=PC41400_RS24335] [protein=Rrf2 family transcriptional regulator] [protein_id=WP_042230469.1] [location=complement(5369912..5370367)] [gbkey=CDS]
MNSEFTIAVHSLVLLAHLPEHMASSDDIASNVCTHSARIRKIMGCLRKAGFVRTKEGCGGGFILNHDPDEITLGDIYQLT
SLGTLKPNWCSGDQGNDCLVSANMQNVMDNMLDGAEQNMIQYLKGVTIGSMLRTLNETEKKAADAEKNLIS
>lcl|NZ_CP026520.1_prot_WP_042230470.1_4830 [locus_tag=PC41400_RS24340] [protein=Maf family protein] [protein_id=WP_042230470.1] [location=complement(5370454..5371080)] [gbkey=CDS]
MSQSGSAPYEQGIKLILASSSPRRQELLKGMGLNFEIISSDADETVEESLSPREIVETLAVRKAEAVSVSLPVSYAPDRT
VVIGSDTIVVLGGEVLGKPRDERDALRMLSALQGQTHEVFSGVACLSADRRNVSVSHRRTKVTMKAMDEQRILRYIGTGE
PSDKAGAYAIQGIGATLIEGIEGDYFNVVGLPVSLLSEMLEPFGIRVI
>lcl|NZ_CP026520.1_prot_WP_009672966.1_4831 [locus_tag=PC41400_RS24345] [protein=DUF4321 domain-containing protein] [protein_id=WP_009672966.1] [location=complement(5371133..5371375)] [gbkey=CDS]
MKKNGFTLVLFLVIGLVVGIIVGQLLAPVKALAFLTKTTDIVWQPKADLQVIQYDILLQIKLNLCSIIGLAAAFWLYRKV
>lcl|NZ_CP026520.1_prot_WP_042230472.1_4832 [locus_tag=PC41400_RS24350] [protein=N-acetylmuramoyl-L-alanine amidase] [protein_id=WP_042230472.1] [location=5371525..5372733] [gbkey=CDS]
MKKTTKLAGLTLALLLVFPYEAMAYKVVVDPGHGGKDPGAVGVNGLNEKTVNLDVSLKLRDELKKKGVDVVMTRSDDRYL
SLDERVQFTNAQNADVFVSVHANSNPSSAANGSQVLYYDSRYPQDDYPASPEMEALTPSSRLLAQKVLSSVLSQTGLKDQ
GLVPSAVYVVRKGKIPSILVETGFLSNKKEADLLADPAFRTKMAKAIADGVVSYQPPVFPDTAGHWAREAILRIKDKGWM
EGIGNNFEPNRALTRAEFVTVMDRVFGFDGLLGTVNSSPAPAFGDLKSTQWSYPVMTKAVRLGLLDGYADQTVRPNQAIT
RGEVSAILQRLREMKGTPTLTPPAGQDVFSDVPASMWCAGAVYTLTKAGIVDGTSPTEFKPNRYITRAEIAVMMDRYSKL
QP
>lcl|NZ_CP026520.1_prot_WP_042230473.1_4833 [locus_tag=PC41400_RS24355] [protein=SPOR domain-containing protein] [protein_id=WP_042230473.1] [location=complement(5372796..5374622)] [gbkey=CDS]
MNKARISYRIDGNDRTNRKEDKSGGKVIPLTKEEFSVSEALWRDAEADRAKKARQNRRDTEAGPRNREGLNEYTADFGTW
QSPYDAETERIAKLIRESEPGYRGDSGYFESTINERFREEKERRQEYEELDRTAYTERRDEEATARQMRDFEGPSEQEGE
YTDYRFREKPSVERGNSGERGREERYPLPPNLGWPEDPSGFIDHGTASLPGYNSLRAAQEDEREDDGYTRSEYNGVQDTR
DRRFRSSSEERSFHDRQLAEAPHSGQRRNPPVYYEGEGADYRREPVYEDSYNRLHWEEPVKSGVRISRSGSGGFLRIATS
VAGAMVTGAAFGYFVLSMFSGGAEKPITIDAGPAATAQAQPSPGAVTPLKPDASAPAASAPSAGTAVAAPPVGGPTAAAS
IPARSYTVLQNGIFSTPQSAETARADARGKGAAAVAEPGTKTTVYVGLTGSKEDAAKLKTALAAKKIEVFAKPLDLPAVG
KVYWKSGSAQPLADYLAQGAKVAGTAGSLSAAKMAEASPGALDAASLQPLDAAAKAWAALQPAVAAGIPDAAKPQVEEMN
AALAKAVAKLGEYGKSPSAPLLAEAQEALMQYVVAEKRLMQQLTAASA
>lcl|NZ_CP026520.1_prot_WP_042230476.1_4834 [gene=murC] [locus_tag=PC41400_RS24365] [protein=UDP-N-acetylmuramate--L-alanine ligase] [protein_id=WP_042230476.1] [location=complement(5375234..5376619)] [gbkey=CDS]
MNTPEHVHFIGIGGYGMSAIAKVMLEMGYQVSGSDLAQQELTEKLRAKGARVYIGHEAQNVNGADLVVYSTALSKDNVEM
VAAGEQNIPVIHRSQMLARLMNERKGIAVAGAHGKTTTSSMIALVMEHCGTDPTYIIGGEIMNIGSNAKAGKGDYVVAEA
DESDGTFLQYHPTLALVNNIEADHLENYDGDFENLKKAYAQFLSQVREDGKAIICLDDEYLRGMLPQIKSETLTYAIDRD
ADYVATDIELGDRKVSFSVRRGGETLGKLRLSVPGRHNVYNAMATLITCLEAGLTFDQIAEAIHEFRGAKRRFQVLGEVN
DILVIDDYAHHPTEIEATISAAKATGKHLIAVFQPQRYTRTFYLFEQFSHAFKEADEVLITDIYSPAGEKQIEGISSEKL
VELIRKNSNPNVTYIPTRGEAEVYLTDRVKPGDLVITMGAGDIWKTADGLAKTLRARYEAE
>lcl|NZ_CP026520.1_prot_WP_042230478.1_4835 [locus_tag=PC41400_RS24370] [protein=folylpolyglutamate synthase/dihydrofolate synthase family protein] [protein_id=WP_042230478.1] [location=complement(5376628..5378004)] [gbkey=CDS]
MEQYTEEASPFKTYTEAVNWITGLVPVGIKPGLKRMDLLMEKLGHPHRRLKFIHVGGTNGKGSTCAFLTKVLIRSGYDVG
TFTSPYLEKFTNRIQYNGQDIEEEHLLLLANRLKPIADELEETELGSPSMFEICTALALLYFGTVTFPDYVVLEVGLGGR
LDCTNIVSPVISVITNIGHDHMDILGDTLEQVALEKAGIIKPGVPLVTTVEQPELAELFIRDAKSKRTTVYLLGKQFTFE
PVSTELDGQKLNFTGPFRSIENVPVSLNGAHQLKNAAAALMTLEVLRQYNALIVDDEDLFAAMSETQWPGRLEMVSDNPR
ILLDGAHNPEGAQTLAAALNDTYSYKKLHLMAGMLSTKNQPGYFRHILPMVDTLILTEPDFHKKENASRLADRARETLRE
LDVNRDIDIVVEPDWKKALEQLKQRTGQDDLAVVSGTLYLISDVRSWLLYTTDSEKGW
>lcl|NZ_CP026520.1_prot_WP_042230479.1_4836 [locus_tag=PC41400_RS24375] [protein=valine--tRNA ligase] [protein_id=WP_042230479.1] [location=complement(5378084..5380750)] [gbkey=CDS]
MAETSDKNSVSMPTTYDPSQSEKKWYDYWISNGFFRAGQRPDAETYTIVIPPPNVTGMLHLGHALDFTLQDILVRTKRMQ
GYDALWLPGSDHAGIATQTKVEQKLREEGLSRYDLGREKFLEKVWEWKDLYAETIREQWAKMGFSLDYSRERFTLDEGLS
KAVREVFVSLYNKGLIYRGKYIINWDPKARTALSDIEVEYKEIQGSLYHLVYPLKDGSGSITVATTRPETMLGDTAVAVH
PEDERYKHMIGKMLVLPVVGREIPVIADEYVEKDFGSGAVKITPAHDPNDFEVGKRHDLPQILVMDESGTMNVNAGPYQG
LDRADCRKKIVADLQEQGVLVKIEEHVHQVGHSERSGAVVEPYLSTQWFVKMQPLAERAIEAQKNGTGVNFVPDRFEKIY
LQWIENVRDWCISRQLWWGHRIPAWYCADCGEVTVAQQDAHACEHCGSENIRQDDDVLDTWFSSALWPFSTLGWPEQTED
LQRYYPTNVLVTGYDIIYFWVARMIFSGLEFTDQVPFKDVLLHGLIRDAEGRKMSKSLGNGVDPLEIIEKYGADAMRYML
STSSTPGQDLRFRIERVEQTRNFANKIWNASRFALMNLEGFTAADIDLSGKLGTADRWILHRLNETARDVTRLIDQYEFS
ETGRLLYNFIWDDLCDWYIEFSKLSLYGEDESAKKTTKSVLAYVLDRTQRLIHPFMPFISEEIWQHLPHEGDTITLAAWP
VYDAQFEAPEAVREMELLMDIIRAVRNIRAEVNVPMSKKIELLVKPSGQDVETILSRNEEYLRRFCSTSELSISAELASP
DKAMTAVVTGAELFLPLAGLIDIAQEVARLEKEMQTLIGEVTRIEKKLSNEGFIAKAPAKVIEEEKAKMADYADKRDKVV
ARLAELKA
>lcl|NZ_CP026520.1_prot_WP_009672975.1_4837 [locus_tag=PC41400_RS29975] [protein=hypothetical protein] [protein_id=WP_009672975.1] [location=5381249..5381413] [gbkey=CDS]
MQKWWWKTRFTVRGVLFPLICIQFVRTLIFPNGLDVFLLFVLFLAYLGFLLDVY
>lcl|NZ_CP026520.1_prot_WP_129112532.1_4838 [locus_tag=PC41400_RS24380] [protein=hypothetical protein] [protein_id=WP_129112532.1] [location=complement(5381483..5382457)] [gbkey=CDS]
MKKLNKALVAALLLALLSPAFQPGQANAQAAMNQNARASFDQLANNVWGFLNEYLLPILTGGNQKFPVPKPEPKPDPTPV
PTPVPTPTPDSGNGKGQNNGGGKGNGNGQDKGNNGNNGQGNNGNGQGNNGNGQGNNGNGQGNNGNGQGNNGNGQGNNGNG
QGNNGNGQGNNGNGQGNNGNGQGNNGNGQGNNGNGQGNNGNGQGNNGNGQGNNGNGGKDRDDRYDSINKETIRKLNALKA
ACQRDLVSYAMQFAQATTTHQKSTIYKEASERFAQCVSDFKDIVGSAQNTLQNEGFSTDIIQEYQREYDIQMETGRMLLD
NIVK
>lcl|NZ_CP026520.1_prot_WP_042230482.1_4839 [locus_tag=PC41400_RS24385] [protein=hypothetical protein] [protein_id=WP_042230482.1] [location=complement(5382501..5383193)] [gbkey=CDS]
MNKRRRLFLWVGTGILLAGMAGSFAAVRISDEMIRTAASLSDGQADAEADAAGKEAPEDHPVPGGGAPASPGADAQDGSR
PQEAFPGVKAAPGSGKAPASSAPPARRDPGKAGHQAAGSAEDGDGRGSGSAKSRSKDPAAISADRAERLSEEISWSERMK
ITTVLLKRLQASDIKQLSLLMKDGVSREEKKKAKAIIMARLSESEYNELIAIAGKYGLSKGKSYQQSVNE
>lcl|NZ_CP026520.1_prot_WP_042230484.1_4840 [locus_tag=PC41400_RS24390] [protein=VanZ family protein] [protein_id=WP_042230484.1] [location=complement(5383190..5383681)] [gbkey=CDS]
MLWTLLVFHWSHQSFREQSIQPLLHKLFTKEEIHDYIPQVTVRYHHTIIRAQSDPMRFIEFIFRKTSHLVVYAVLAALAY
QALCFWGSRRRKRILFSLLYVVVIAMGDEWVQSAAAMRTSAIQDVVLDTAGGLAGLSAAVWAVPWGIRLWRTSNRIKRRE
AGR
>lcl|NZ_CP026520.1_prot_WP_042230486.1_4841 [locus_tag=PC41400_RS24395] [protein=UDP-glucose/GDP-mannose dehydrogenase family protein] [protein_id=WP_042230486.1] [location=complement(5383704..5385056)] [gbkey=CDS]
MEKITVVGSGYVGLVSGTCFSEIGNRVICCDVDPFKIAMLRKGEIPIYEPGLKELVKKNVDADRLFFTSEIGDAIEASDI
IYIAVGTPMGDNGEADMRYVHEVARTIGKHLNSYKIIVNKSTVPVGTGEQVRQIIMENRKNRFVQFDVVSNPEFLREGSA
IEDCMNMERAVIGATSDKAAKRIADLHAPFRTRIFQTNLESAEMIKYAANAFLATKISFINGIANVCERVGADVTSVAAG
MGLDSRIGAKFLQAGIGYGGSCFPKDTFALSYIAEEAGFDFSLLKSVIAANDEQRLVVVDKLKQALGCLEGKKIGVLGLA
FKPNTDDMRYAPSLTIIPELVRLGAYVRAYDPIAMQAARNQIAEYYEDFETLEEALEGCDACLILTEWSEIVEMDLARVK
QLLTAPIIVDGRNCFSVREMKEREFVYYSVGRPAVTTGSANASAAKTVSL
>lcl|NZ_CP026520.1_prot_WP_042230487.1_4842 [gene=galU] [locus_tag=PC41400_RS24400] [protein=UTP--glucose-1-phosphate uridylyltransferase GalU] [protein_id=WP_042230487.1] [location=complement(5385058..5385969)] [gbkey=CDS]
MKVRKAIIPAAGLGTRFLPATKAQPKEMLPIVDKPTIQYIIEEAISSGIEDILIISGRGKRAIEDHFDKSFELEETLDKK
GKHGELKQIQAISDMANIHYIRQKEPKGLGHAIWCARSFVGNEPFAVLLGDDIVQSEEPCLKQLLRIHSRYSSSVVGVQR
VSDEDVSKYGVIAPRGSSIEPDVFFLETLVEKPSRKAAPSNYAIMGRYVLTPEIFEILENQAPGAGGEIQLTDAIRRLNE
QKPVVAYNFSGVRYDVGDKFGFIKATLDFALQREDLKQDLLGYIQSVHEQHAALHNPISKVAK
>lcl|NZ_CP026520.1_prot_WP_042230489.1_4843 [locus_tag=PC41400_RS24405] [protein=aminotransferase class I/II-fold pyridoxal phosphate-dependent enzyme] [protein_id=WP_042230489.1] [location=complement(5386008..5387246)] [gbkey=CDS]
MAVQLQTHASPARAAQLQTGTAPGTADSLPGTGRSRILLSPPHIGEEEQKFVREAFETNWIAPLGPNVDAFEAELAAYTG
SKGAVALSSGTAGIHEALRLLGVGPGDRVLCSSLTFVASVNPIAYQGAEPVLIDSDEATWNMSPAALERALAESKREGRL
PKAAVVVNLYGQSADMLPLLELLDAYGVPVVEDAAESLGATCSGKSSGTLGRFGVYSFNGNKIITTSGGGMLVSDDMEAL
EKARFWATQARDQAPHYQHSEVGYNYRMSNILAGVGRGQLRVLDERVNARRAVFDRYREALGGIEGIAFMPEAPYGRSTR
WLTALTVDPAAAGVTTEELIARLAAENVESRPVWKPMHLQPLYGKCAYYPHEEGVSVSDRLFEQGLCLPSGSSLSESDQD
FVIGILLHALNA
>lcl|NZ_CP026520.1_prot_WP_042230491.1_4844 [locus_tag=PC41400_RS24410] [protein=acetyltransferase] [protein_id=WP_042230491.1] [location=complement(5387535..5388170)] [gbkey=CDS]
MRRLLLIGDGGHAKVVRDIARAGGQYELAAVLDDRYAVPADGADGLPRGPVALAARMLAEDEDLRVFVAIGSGTVRRRVA
GSLGIEPGRFALLIHPRAVVSGDASLAPGTAVMPGAVVNAGARIGAHAIVNTGAVVEHDCVIGDYAHVSPNAALAGGVDV
GAGAHIGIGASVIQGLAVGEWSTLGAGAVAVRDIPAGCTAAGVPARVIRQA
>lcl|NZ_CP026520.1_prot_WP_275938057.1_4845 [locus_tag=PC41400_RS30450] [protein=sugar transferase] [protein_id=WP_275938057.1] [location=complement(5388167..5388781)] [gbkey=CDS]
MLRTADLAIALPLVLLLLPVYAVVGLLVRMKLGSPVLFKQKRPGRFGRPFHVYKYRTMTDKRDSFGNLLPDEVRLTAFGK
GLRKLSLDEIPQLLNVIKGDMSLVGPRPLLMEYMELYTDEQKRRHWVRPGITGWAQVNGRNAISWEDKFKLDVWYVDNRS
FRLYVKILLLTAVKVVRREGVSQEGQATTTKYTGSAASGIGSPS
>lcl|NZ_CP026520.1_prot_PC41400_RS30455_4846 [locus_tag=PC41400_RS30455] [protein=glycosyltransferase family 4 protein] [pseudo=true] [partial=3'] [location=complement(<5388959..5389900)] [gbkey=CDS]
MSVLQQAGHEVHLISSPEGYDEQAMKNSGLKLKFVQMNRKIRPVEDVRSILRMRRLIKREGYDIVHTHTAKAGIIGRAAA
WLARTPVIMHTSHGLPFYEGQSKTAYTVYRTLEKIGTYFCDALASQNREDITKLQSLNAGKIVYYEGNGVDLDKLDKAAG
TISREELESLKATYGVPENKTVLLMGARMEPVKDHHFLLDALALLKSQGITDFVCLLAGKGPLEKQILERIAELGLQKEV
LLIGHRSDLYAFLKMADIVLLTSEKEGIPRFLMEAMAFSKPVVASDVLGTRELVRHEDTGLLVPYKNTGALAKA
>lcl|NZ_CP026520.1_prot_WP_042230493.1_4847 [locus_tag=PC41400_RS24420] [protein=NAD-dependent epimerase] [protein_id=WP_042230493.1] [location=complement(5389997..5391010)] [gbkey=CDS]
MSILITGSAGFIGYHLALRLLESGAKVVGYDNLNDYYAVTLKQDRLKQLQNYPNYHHYTADLCDHEQLNRVFGEHEIRVA
VNLAAQAGVRYSLQNPRAYADSNLTGFLNVLEACRTYKIDHLIYASSSSVYGANTKMPFSTDDPVDHPVSLYAATKKANE
LMAHSYSHLYGIPVTGLRFFTVYGPWGRPDMAYFSFTRNILEGKPIQVFNEGKMRRDFTYIDDIVEGIVRLLDRPPAPNK
DWDRENPDPSTSYAPYKVYNIGNNSPVELMTFIRTLEESLGTRADIEFKPMQPGDVQATFADIDALQRDVGFRPSTPISE
GLRKFADWYVGYYGSGS
>lcl|NZ_CP026520.1_prot_WP_042230495.1_4848 [locus_tag=PC41400_RS24425] [protein=glycosyltransferase family 4 protein] [protein_id=WP_042230495.1] [location=complement(5391015..5392202)] [gbkey=CDS]
MNVLWAHDHTFYFTDSGKFYSSGKLPYAVWERYLSVFDSLTVVSRGKRMTSAHEMEKKSLSSGAGVNFIVLPSLSNPVNK
ITKKGYVERVLTEAVMKADAVIARLPSEIGSEAIAIAKKLGKPFAVEVVACAWDGLWNYGSVQGKLYAPIATWKTKSDVK
EAPYAIYVTEQFLQKRYPCPSGRTQSCSNVELPEMGEDVLQRRLERIRSMKTPVTIGLIGSLNGKTKGIDTALNALARIK
DQLPPFKFRILGDGDKSRWIAMAGSLGLQENVQFDGVLPSGSAVFGWLDSVDIYIQPSFQEGLPRATIEAMGRGCPVIGS
TAGGIPELLDPGCLFKPGQDKQLAALLTGTVINRDWMLEQASRNFAKAANFTKTKLDRDRSEFWHSFHQYVSRGV
>lcl|NZ_CP026520.1_prot_WP_042230497.1_4849 [locus_tag=PC41400_RS24430] [protein=acyltransferase] [protein_id=WP_042230497.1] [location=complement(5392180..5392782)] [gbkey=CDS]
MRGRDRFHKFKTVLDIVIRILRLFPRPALKYAWTLTDLLPDTAGVAMRYCLLKCLTLSCGDNVFVGRSVEIKYWERLRIG
SNVSIHKQCYLDACGELQIDDDVSIAHQSSLISFQHSWENESLPIRDNPVVCSKIHIHRDVWIGCGCRILAGASIGSRSI
VAAGAVVTRPVPSMTVAAGVPAKIIKSIRKEGHYERSVGT
>lcl|NZ_CP026520.1_prot_WP_042230699.1_4850 [locus_tag=PC41400_RS24435] [protein=hypothetical protein] [protein_id=WP_042230699.1] [location=complement(5392694..5393956)] [gbkey=CDS]
MIGYVLLMLLLEKGIFQLDERMGIISFFVLLLPLFCMIRWPDQPFLLYIGFCVMLIGKFVYALTATPLAGPDENHYYEQV
VTYLGLGDFLNYAFEHISTYLFNSSAYPIFGLMYMPFFKFLDVSDPLVIITYNSVMLIWIAYLIYALNRSFFGYEQANRR
MYEGWIILGLFVSPSFMMMTSLFAKDVTCVALGLYCTHLLLKRKYVLFLLVMLYATGLRDYAIVYTLCFYLLFTKRFKTA
VAMLVVSAGVLAVKIGGLGIVNAVLLTAFLFLSPNPVNPENWETNVMYRSMEAVAMLVALAFAVLMFIRYKETRAFYRIV
VVLLFAYACTLVLVGYMTVTGRDLEYGVGTIGDNMVRKKLPILPLLYMFQAYTAGWTLKWLKSIRDKRRGIHERPRPVSQ
IQNGAGHRHPHSPSLPEAGA
>lcl|NZ_CP026520.1_prot_WP_042230498.1_4851 [locus_tag=PC41400_RS24440] [protein=glycosyltransferase family 1 protein] [protein_id=WP_042230498.1] [location=complement(5394006..5395139)] [gbkey=CDS]
MKPKTKILHVVGKMHPGGIETLLMNLYRTIDRDKYEFHFAVQFEEKGFYDDEIIELGGRIFRQPHPKKGLRAFRRAFLAN
LKEHGPYQVVHSHIYQFSGYVLKLAGEMNVPVRISHSHNTHDSKRESWLRNLYRRYMRGLIRSNSTTMLGCSRSACEALF
GPDCWGDERVVVFPNSIQVQPYGQLERDRSAYRRAVGVTDPHVPVIGHIGRFSEQKNHRFLLDIFARFLRTRPDAQLVLI
GDGPLRSEMEARVRELGIADHVHLLGVRKNIPELIGAFDLFLLPSLYEGLGIVVIEAQAAGVPCLISDQVPAEADLNLGL
VSRLSLNESADTWAGAIRQKTVGSRLDWPVRLEALQECGYDIQTSVRRLERMYAAAP
>lcl|NZ_CP026520.1_prot_WP_042230500.1_4852 [locus_tag=PC41400_RS24445] [protein=polysaccharide pyruvyl transferase family protein] [protein_id=WP_042230500.1] [location=complement(5395129..5396352)] [gbkey=CDS]
MKHILLAGVPHSTNLGDGLIAMTVNQLIRTCGEYTVTNFDLTAGLYRETPVRSGSVQRSDGGPALTREYKLNEVSVKKKL
TPDFLRYMKSYSVHKRKDAVLGEQIRELVTQADAVLIGGGHLFIDTYLSFPIAMRRLVREVKRQGKPLHVVFVGMRGPWS
FLARKWFTDILHYAKTISVRDEDSLAYLLSIAPSLAHKAVALSDPALVTLEAFPGKRAAAIAKLPSVRTVGLGIMDPNEM
RRHSSYVWERDDCADWWHSLAKGATDGGSRVRVFTNGADTDNAFVERYIKPRLAGMPGVEFAPYPGDVGDLVESIRDCDT
VIAQRLHACIPAISFGKPTYGIAWDKKLASIFKDLGLSGHVLDFRTDPKAVLAGLVPEKSRQEGSVRIVNQKKRELVTYV
GRILHEA
>lcl|NZ_CP026520.1_prot_WP_241688728.1_4853 [locus_tag=PC41400_RS24450] [protein=flippase] [protein_id=WP_241688728.1] [location=complement(5396354..5397706)] [gbkey=CDS]
MSIHTTAKPLTQRSLSKMLMKSSSLSFLVNVGGIGIALIMQILLARQLGAVSYGYYSFVTTAVTFLVFPTKLGFDTAIVR
YVSSYRAHGQWAEVKGLLRRANQITAGLSLAVTVISLLVVLLMSGSMNGELLYTYIAGLLGIPVMALALLRQSSLQALKE
MLHAQLPEKIIRPVVFIALLFLYNAIADTKAGAGEAMLLFLAAMAVSYAIGAWVLKRRLTPKIQGAGTAYNTREWVNVSL
SLMFVSGMYLVLGQLNVLMLGWMEGTTEAGIYGAAVRIATLVSFGLTAINMTAAPFISESYAKKDDKQLRHVLSLSNLAG
FLFAGVIYVAFLFLNDPILGLFGEEFKRGSLALLILSTGQLVSAFSGQTGTLMTMTGQQKTLGRILAGSAVLNIVLNFVL
IPAFGMLGAAVTNLLCTVCWKTAMVIVIRRKLGISIGMFAWMTANKKRGN
>lcl|NZ_CP026520.1_prot_WP_042230501.1_4854 [locus_tag=PC41400_RS24455] [protein=nucleoside-diphosphate sugar epimerase/dehydratase] [protein_id=WP_042230501.1] [location=complement(5397737..5399572)] [gbkey=CDS]
MSYSKRTPMLVLIDMIIVWASIYLTFLAHFRGGIPQEVMSVIMLYTLVSGISLACAMFYFQLYNRIWQYASVGEMVAIAK
AVIVSCLISYAVTSFLSAGEITLYMFLQNLNGILLMLGGSRFVWRVFCDNFYGSSRSRGGKPAQRTLVVGAGSCGVMFVK
ELAHNDSFMMKPVAFVDDDPYKQKLQLHGIPVMGNRLDIPRIVEEQRIDDIIIALPSVSKTQISDIINICKTTKARLKII
PLIQEYIQGKMAANQMRDVQVEDLLGRDPIQTDLQNIADYVQNKIVLVTGAGGSIGSELCRQVAPFKPARLLLLGHGENS
IYGIEMEMRRLFPNLPIETVIADVQDRTRIADVFREFGPQVVFHAAAHKHVPLMERNPSEAIKNNVFGSKNVAECADEFG
AERFVLISTDKAVNPTSIMGTTKRIAEMFIQSLDKHSKTKFVAVRFGNVLGSRGSVIPRFKDQIARGGPVTVTHPEMVRY
FMTIPEAVQLVIQAGAFAKGGEVFILDMGKPVKIVDLATDLIRLSGYEPNVDIDIEFSGIRAGEKLYEELLTSEEGMSST
IHDRIFIGKPMNINRTELEFEMRRLERVLGSDPEEIRALLQHLVPTYRNVS
>lcl|NZ_CP026520.1_prot_WP_042230704.1_4855 [locus_tag=PC41400_RS24460] [protein=CpsD/CapB family tyrosine-protein kinase] [protein_id=WP_042230704.1] [location=complement(5399657..5400340)] [gbkey=CDS]
MSPSNKQRPIIAHSNPMSPISETYRTLRTNIQYSGVDGEIRTLMVTSASPWEGKSTTIDNLAVVYAQTDKKVLLIDADLR
KPTAHRTFGLSNRTGLTDVLTRQAEVEEALKETDIPGLYVMTCGTLPPNPSEILGSLRMKELLERLKQEFDVILVDTPPV
LAVSDAQVVATQCDGVVLVVNAGGVKRELVQKARDSLQFVHARVLGVVLNKVARKDAKMNTYYDYSS
>lcl|NZ_CP026520.1_prot_WP_042230503.1_4856 [locus_tag=PC41400_RS24465] [protein=Wzz/FepE/Etk N-terminal domain-containing protein] [protein_id=WP_042230503.1] [location=complement(5400324..5401079)] [gbkey=CDS]
MSIDFEIKRYLAVLRKRLWLILLIVAAGCLITGVVSKYYLKPVYQASTKLIVNSPADTQGMLRLDMNAINTNISLIHTYK
EIIKTPAIMDLVVKEHPEYGVTSDDLMRRIQFSSVSDTQIFTLSLEDASYSKAAQMINTVAQMFQKQIPNIMKIDNVFVL
HEADPAKTPAPVKPNVPLNVAISFLASLMFAIGLVFALEYFDDTLKTEEDVEHYLGLPTLVAVPEIRAGDMKPKRTPIGK
KVGEASGVPIQ
>lcl|NZ_CP026520.1_prot_WP_042230505.1_4857 [locus_tag=PC41400_RS24470] [protein=tyrosine protein phosphatase] [protein_id=WP_042230505.1] [location=complement(5401228..5401995)] [gbkey=CDS]
MIDIHTHILHGLDDGPGSLEQSLEMARMAVSNGISTVFATPHHRTSRYYTPSADIEEAVAGLNESLLEHGIPLHVKSGQE
IRVTPNLINDWYEDTLLTLGGTRYLLIECPAYSLPAELAEICHECSVLGLVPVIAHPERNAEVAGDPDKLLQLAEWGAVF
QLTSDSILGTFGSKVQSLAFELCSRRLIHLVASDAHDLVRRPFRLSEAYEKIEAKIGGDYAEYCRRNAIRVANDLALEKR
EPVAGRPRRIFAMFR
>lcl|NZ_CP026520.1_prot_WP_241688729.1_4858 [locus_tag=PC41400_RS24475] [protein=LysM peptidoglycan-binding domain-containing protein] [protein_id=WP_241688729.1] [location=complement(5402977..5404467)] [gbkey=CDS]
MATLTEEQNGLRFDIYERVHLAEGAAGIRDLDDVELIPSIEVIPQQEQALLKGSLLLNGSYLDEQDTKRSLEHSIPVEIT
LPLSRVQRIEDILVDIETFDVDVLNSRSLNVTGVLSLHGVEMAYEAGGWSEEDEEVVFVHEAASAPQNAPAEQWSEPAGG
ADELELRQQTEYSASADGQAKLQPPTTSAQPQYQAQPQAPVYPGSAYKAENPWLRETRTEQPEVQAAAPSAWDRLATPKS
PPPQEDSSASYADRSFGELTLHQLSSERSYRSAGVEEAQEQRSPSAAAGQQPLAQAADPASASREEQEAAGQAEREETAP
AFSVDAEVIEPERDPVPASAPAQESLEVFADEEPAVQEETKELKVAFGSKKQQDSPSSGGSGLQSLFANGAGSSASSYEH
TEKDEREYALASGSEESKADALEWKKLFIQTEKEDQQFRKVTMCIVQKEDSLDLIAKRYEINPRELQLYNRLEDQEVSPG
QVLYIPGKAVFSSSDQ
>lcl|NZ_CP026520.1_prot_WP_042230509.1_4859 [locus_tag=PC41400_RS24480] [protein=RluA family pseudouridine synthase] [protein_id=WP_042230509.1] [location=complement(5404570..5405433)] [gbkey=CDS]
MTGRRKGEWIELALPGDTCAWEEVNAVLPVPLKLWRKLEREDGIRRQGKKVLLKVFPDEPSPVPGEWTDLQVLFEDDYLL
VVNKPAGMAVHPSEPGQRGTLANAVAGYYESTGQSCGVRHIHRLDQDTSGAVLYAKYELPHLLLDEAMREKAIGRIYAAV
CGGLPAPPAGTISAPIGKDRHHRARRRVSPGGDAAVTHYRTVESFRDAALLELRLETGRTHQIRVHMSHIGHPLLGDELY
GGRMNAFGRQALHGAHLLFRHPVTGEELAVDAPYPDDFRELLAALNA
>lcl|NZ_CP026520.1_prot_WP_164977153.1_4860 [locus_tag=PC41400_RS29980] [protein=hypothetical protein] [protein_id=WP_164977153.1] [location=complement(5405402..5405545)] [gbkey=CDS]
MKRYYFFDFPIDLIFIVFVFVVLLAVLAVVLRKGGRYRDRQKKRGMD
>lcl|NZ_CP026520.1_prot_WP_042230511.1_4861 [gene=hemL] [locus_tag=PC41400_RS24485] [protein=glutamate-1-semialdehyde 2,1-aminomutase] [protein_id=WP_042230511.1] [location=complement(5405835..5407136)] [gbkey=CDS]
MELNHKRNDTKSAAAFAEAKQVIPGGVNSPVRAFKSVGLTPMYMERASGSRVYDIDGNEYIDYIGSWGPLILGHAHPEVI
EAVKSTAEKGTSFGAPTLIETEMAKLVAERVPSIDIVRMVNSGTEATMSALRLARGFTKRNKILKFEGSYHGHADALLIK
AGSGVATLGLPDSPGVPESVAQNTLTVPYNDLETVKVVFEKFGEDIAAVIVEPIAGNMGVVPPLPGFLEGLREVTSKFGA
LLIFDEVMTGFRVHLNCAQGLYGVTPDLTCLGKVIGGGLPVGAYGGRKDIMEQIAPSGPIYQAGTLSGNPLAMAAGYTTL
KLLGQPGVYEELERKSARLAEGFLANAKEIGVPSTLNRIGSMVCPFFTEQKVVNYDTAKTSDLSVFNRYFGHLLDLGVSV
APSQFEGMFVSAAHTDEDIEVTIEANREALKRL
>lcl|NZ_CP026520.1_prot_WP_042230513.1_4862 [locus_tag=PC41400_RS24490] [protein=hypothetical protein] [protein_id=WP_042230513.1] [location=complement(5407193..5407459)] [gbkey=CDS]
MDEVDVTQLAALRTITKPKPLADKKQKLEANACGAAAAKAVYLLKQSHAEACIIETTDKQKPGRLCSFEASRLLSRREGA
DRGLPASA
>lcl|NZ_CP026520.1_prot_WP_042230515.1_4863 [gene=hemY] [locus_tag=PC41400_RS24495] [protein=protoporphyrinogen oxidase] [protein_id=WP_042230515.1] [location=complement(5407518..5409008)] [gbkey=CDS]
MNEARKIIVVGGGITGLTAAYYVRKRLKEAGCTADVTLVEGAEKLGGRIHTLRKEGFVIEKGPDSFLARKLPIIELSREL
GLEGELTGTNPEAKKTYILRDGTLHPMPAGLVLGIPTDIEPFMRTGLVSAKGKARAMMDLVLPKKEDDSDESLGHFLHRR
LGKEVLDQIVEPLLAGIYAGDTYALSLQSTFPQFGEMEQQYRSLIYGMISNRKKSQQEALNLPAPGRNSVFLTYKNGLQT
LVERLEEVLREEGVTIRTGTKVTELRKEGELYEVGYEGGQRETADSVILTLPTYHSAELLGDHPSVDKLRAINYVSVANV
VIAYREKDANVKFDGSGFLIPRSEGRSITACTWTSTKWLHTAPEDRVLLRCYVGRSGEEDWVHLTDEEIVAKVRKDVAEL
MGITAEPLFYEVTRLYRSMPQYPVGHLDHVRELRSDLAERMPGVYVTGAGFHGVGLPDCIRQGREAAFEAVDRLTGAADE
SAAQASAEGTKTAAKV
>lcl|NZ_CP026520.1_prot_WP_042230517.1_4864 [gene=hemH] [locus_tag=PC41400_RS24500] [protein=ferrochelatase] [protein_id=WP_042230517.1] [location=complement(5409110..5410054)] [gbkey=CDS]
MSGKKVGVLVMSYGTPQSMEDIEAYYTHIRRGNKPSPEQLHELTSRYEAIVGGVFPLRENTDNQVAGLQNALASEHDDVE
FVCFQGLKHAYPFIEDGIEQMAQCGITEGVGIVLAPHYSTMSIGSYNKRAREKAAEYGISMTFVDSYHLHPKLIQALTER
VERALGKFGEENRSSVRVLFSAHSLPEKILELGDPYPEQLLETSRAIADKAGIENWQFAWQSAGQTATPWLGPDILDVLR
TLAKEDVKRVLVCPIGFVSDHLEVLYDLDIEAQQLAKEIGLHLERTDSLNTDQLYMQTLSDAVWKQYGELTKQA
>lcl|NZ_CP026520.1_prot_WP_084706607.1_4865 [gene=hemE] [locus_tag=PC41400_RS24505] [protein=uroporphyrinogen decarboxylase] [protein_id=WP_084706607.1] [location=complement(5410058..5411122)] [gbkey=CDS]
MINNDRFIRACRKQDVDRLPVWYMRQAGRYDPDYRKIKEKYSLLEICEQPELAAEVTLMPVKKLGVDAAILYSDIMNPVA
SLGIQFDIVKDVGPVIENPIRSREDVMKLKPIDVEGDLSHVIETIRILDRELTVPLITFAGAPFTIASYLIEGRPSKSYI
RTKELMYGNPEVWHALMDKLGDMVITYLRAHIAAGAKAVQLFDSWVGALAPEDFRVFVLPTIQRIFRELGDLKQPKIYFP
GVSSGELLPLLGDLQADVIGIDWRVSIPEGRRRTGGRFAVQGNLDPYVLTAPMPVIQEHAKRIIDQGITEPGFIFNLGHG
LFPEASLDKLKELTDFIHAYSAEALAGRNAGKGV
>lcl|NZ_CP026520.1_prot_WP_009674217.1_4866 [gene=hemB] [locus_tag=PC41400_RS24510] [protein=porphobilinogen synthase] [protein_id=WP_009674217.1] [location=complement(5411463..5412458)] [gbkey=CDS]
MSFPLVRHRRLRGSAAIRNLVRENALSVNDLIQPVFVEAGHNIKSEIPSMPGVYHFSLDRLEEEIREISELGIQGILLFG
VPETKDAVGTSAFHEEGIVQQASRLIKKINPNLLIIADTCLCQFTDTGHCGVVHHNPETGCTDIANDESLELLVKTAVSQ
AKAGADIIAPSNMMDGYVHAIRAGLDAEGLEHVPIMAYSVKYSSAYYGPFREAVDSAPQFGDRKTYQMDSANSREALREA
ESDVLEGADFLMVKPALAYMDILQLLKQNFDLPLVAYNVSAEYSMVKAAAQNGWINEKEIVLETLLGFKRAGADIILTYH
AKDAARWLKNV
>lcl|NZ_CP026520.1_prot_WP_042230521.1_4867 [gene=cobA] [locus_tag=PC41400_RS24515] [protein=uroporphyrinogen-III C-methyltransferase] [protein_id=WP_042230521.1] [location=complement(5412554..5414116)] [gbkey=CDS]
MERGKVYLVGAGPGDPKLITVRGLQAIERADVIVYDRLASPRLLKHRKPDAELVFVGKLPDKHMMKQEDINRLLVDLALQ
GKVVTRLKGGDPSVFGRVGEEAELLADHEVPFEIIPGITSAIAVPAYAGIPVTHRDFTSSVAIVTGHEYPNKTYTKLDWE
HLAKGIGTLVFLMGVANLEQISEQLIRHGKPADLPVALIRWGTWTEQQTLTGTLSDIGAKAQEAGFKSPAVIIIGEVVKL
REKLAWFEKKPLFGKRVLVTRARSQASELVAMIEELGGEPVEFPVIKLQPPSSPDAQRLLDEALRGLENYNWVVFTSTNG
VEFFFRRMRELRLDIRGLAGVRIAAVGPKTAEALESRGLLVETLPGEFHADELLVELGKAVRPGERALLPTADIARNALP
QGLRSLGLEVSEVDVYENVPEVEGGEDVVRLLEEGQVHITTFTSSSTVTNLIGALQKLGVQDPVSLLKNTEIACIGPKTE
ATLAHYELAAAYMPPEATIASLVESIAAGRSVKAEEPMGS
>lcl|NZ_CP026520.1_prot_WP_042230524.1_4868 [gene=hemC] [locus_tag=PC41400_RS24520] [protein=hydroxymethylbilane synthase] [protein_id=WP_042230524.1] [location=complement(5414302..5415255)] [gbkey=CDS]
MRTIVVGTRQSALALTQTNQVIDELKELSRQNGIECEFELRKIVTKGDRILDVTLSKVGGKGLFVKEIEQAMMDKEIDIA
VHSMKDMPFELPEGLMVGAVPKRQDPRDVLITKGEKALDELPQGALVGTSSLRRSSQLQHSRPDLRIQPVRGNIDSRLRK
LESEGFDAIVLAAAGMHRIGWQDRISAYVPPEVCVPAVGQGALCIECRADDEFVLNLLSKIQDDDTALAVRAERSFLGRL
NGGCQVPLGAYATVERTGGGTELSLTAMVGSPDGAELLKETLRGSDPEALGREAAEKLIARGADRILAEAYESGQIE
>lcl|NZ_CP026520.1_prot_WP_042230526.1_4869 [locus_tag=PC41400_RS24525] [protein=bifunctional precorrin-2 dehydrogenase/sirohydrochlorin ferrochelatase] [protein_id=WP_042230526.1] [location=complement(5415337..5416092)] [gbkey=CDS]
MKEYPAVLRLEGKRCLVVGGGAVAERKIASLLEAGALVTVISPAATAAVEKWAEAGRITWHCGRYGEADHDYTKDKEGSD
KGRPGTEASFYPDLSPYLIVFAATDNSQVNERVRTDADAAGKLVSSADKPDRGSLIVPAAVRRGKLTLAVSTSGASPALS
AKIGRELQERYGPEYEIYLDFLAELRELVRSRVGDMPLRHELLRRLPSWRVLDRIAAGTYEAWRASLLDRLQAAQGSEEI
FAAVWMEADQE
>lcl|NZ_CP026520.1_prot_WP_042230527.1_4870 [gene=ccsA] [locus_tag=PC41400_RS24530] [protein=cytochrome c biogenesis protein CcsA] [protein_id=WP_042230527.1] [location=complement(5416089..5416931)] [gbkey=CDS]
MFTRSFLYDAIIYLYALSLLFYFSDVARANRSAKRMGTGLLSFVWVLQTAYLGYNLYEHLGRWVFSMFETLFLFSWVLVT
VSLIMSRFFRIELVVFFVNLFGLAVLALNFFSDSNVSPMIDSWNVNDELLFIHVSLAVGSYAAYTISAIFSGMYLFLHRK
LKQKSWTNTMKRLPSLEKIERYTYVSVVIGTPLLMLALALGVVWVLIEKNVNVLFDSKVINSLLVLCAYGFYLFQHHSLS
MSGPKLAKWNLAAFGIVLLNFILSNVFSGFHNWQQTGVGG
>lcl|NZ_CP026520.1_prot_WP_042230529.1_4871 [gene=hemA] [locus_tag=PC41400_RS24535] [protein=glutamyl-tRNA reductase] [protein_id=WP_042230529.1] [location=complement(5416968..5418365)] [gbkey=CDS]
MHIVVVGLNYRTAPVAIREKFTFTQEELSQALVELRQTKSILECVVVATCNRTEIYAVVDRVHICGHYIRKFMEEWFEIP
RTEFNNHLYIYEHDRAIEHLFRVASGLDSMIIGETQILGQVKDAFLEAQRLGATGTIFNTLFKQAVTMAKRAHSETSIGE
NPVSVSYAAVELGKRIFGSYEGKKILIVGAGKMSELTVKHLYASGAERVDVVNRTYERALELAGKFNGHAYPMERLYERL
AEADIVISSTGSTDYVLTKKQLQAVLPKRKSRPLFMMDIAVPRDLDPDISDLPDVFLYDIDDLEAIVDSHLKERKKEAVK
IEEMIGAEIAAFDQWTKTLGVSPVIHALQAKANMVHQETMESLFKKLPDLDEREMKVIRKLTKSIVNQMLRDPILAIKEM
AAEKKGDEALEMFTKMFALEELMAEQAEAEALGEAKAEAVQTAAQKHAKEELTGGFTMQPALARS
>lcl|NZ_CP026520.1_prot_WP_042230531.1_4872 [locus_tag=PC41400_RS24540] [protein=hypothetical protein] [protein_id=WP_042230531.1] [location=complement(5418511..5419044)] [gbkey=CDS]
MRSRIKSKLILFVLAAGLVAWPLTSLYGYVTKGADKPDAERLLFQVSLFQIELLNRFLQDAARSADTRELDSLKQALYTA
SYTHERLVLAEGSEKLHTLGSLPQLMQYILRLQIGGERALKADEQGVLQESAKLYAQLYEAYGKLMTSSGSVVTSQSEAV
SKNDEALRELLEKKLLQ
>lcl|NZ_CP026520.1_prot_WP_016423520.1_4873 [gene=speD] [locus_tag=PC41400_RS24545] [protein=adenosylmethionine decarboxylase] [protein_id=WP_016423520.1] [location=complement(5419186..5419587)] [gbkey=CDS]
MEYSTFGRHVAVDTWGVDFELLNNAEWLQAQMVDAAEACGATVMSVQSKQFEPQGATVLVLLSESHLSIHTYPERGFAAL
DCYTCGETVDPQLAIDHMISLLKPEKVYAKKLVRGMGEFEVETPEIKQAVLAK
>lcl|NZ_CP026520.1_prot_WP_042230534.1_4874 [locus_tag=PC41400_RS24550] [protein=hypothetical protein] [protein_id=WP_042230534.1] [location=complement(5419995..5420582)] [gbkey=CDS]
MAQRLATEYVKTRLQLTEAEMSKFVQMFADQQITAQVKVIENGNQEFVFQEEAGEEIVLSFERVGKQYISSGSCRVTSLR
LTNLMRKAIAEFKGSAIVKRIYATFVMMYTYENGKVIKIVEIKGQHTNVVYEYHDTLGQLENLFRRTEIEREIDMIQDQI
NHLLDMRNEVPCPSLKGDIDARLTKLTRRLFVLEA
>lcl|NZ_CP026520.1_prot_WP_042230535.1_4875 [locus_tag=PC41400_RS24555] [protein=SdpI family protein] [protein_id=WP_042230535.1] [location=complement(5420887..5421555)] [gbkey=CDS]
MTNSIRKVPKWTWKDTLLLIIAAAPILYAIGVYGKLPEQMAVHFGVSGEPNGYQSKGSFLILMSVLSLAMPLLMKWSPSL
DPKRTNYEKFQGVYDLFRILITVLLSGVFFMTLLTNLGYDVPINFWVPLGVGALWMVVGNYMGRARPNYTFGIRTPWTLA
DEEVWRRTHRLAGPVWMLAGIVFMGVSLFPSVVPAWFGIVVILLSVLVPILYSFIKYRQRKN
>lcl|NZ_CP026520.1_prot_WP_174446816.1_4876 [locus_tag=PC41400_RS24560] [protein=autorepressor SdpR family transcription factor] [protein_id=WP_174446816.1] [location=complement(5421557..5421829)] [gbkey=CDS]
MNDTFKALSDPTRRQILKLLKEKDLSAGEIADHFQMSKPSISHHLNVMKQAKLVLDERKGQSIIYSLNTTVIQDALGWLM
DFTHKDEGGS
>lcl|NZ_CP026520.1_prot_WP_241688730.1_4877 [locus_tag=PC41400_RS24565] [protein=hypothetical protein] [protein_id=WP_241688730.1] [location=complement(5421908..5422600)] [gbkey=CDS]
MNPSTDLKQLQDHHIRLSQLMSPLYLLIPLVFWAFFNYQGVGVTVKPVLLGAAGWIIALALRGPVSLLAARMPAEKGRLL
LVSSSGPLEELVRLVTLLIAGLTIGSAASIGIGWAAIEIVYTLVSLFAVASLSKRTDEKAMQAKAMLEAQGTYGAHPAWG
LIERVFVSAYHIGAALCIARYPWSVAVLIPLHSLLNLAVLAVAKRSLAQAEAVAAFVGTTVLAIGWQLYL
>lcl|NZ_CP026520.1_prot_WP_042230538.1_4878 [gene=yihA] [locus_tag=PC41400_RS24570] [protein=ribosome biogenesis GTP-binding protein YihA/YsxC] [protein_id=WP_042230538.1] [location=complement(5422793..5423431)] [gbkey=CDS]
MKVNQAEFIISAVGPGQYPEDALPEIALAGRSNVGKSSLINRMISRKNLARTSSQPGKTQTLNYYRINDDLYFVDLPGYG
YAKVSKTKRAEWGKFIENYLLKREPLKAVMQLVDLRHPPSKDDQAMYAWLKHMGVPVIVVATKADKVPKGKWQKHAKIIK
DDLEFDRSGPFIIFSSETGMGKDELWEILESAIGWGQAQPDGEVLPESAPSE
>lcl|NZ_CP026520.1_prot_WP_042230540.1_4879 [gene=lon] [locus_tag=PC41400_RS24575] [protein=endopeptidase La] [protein_id=WP_042230540.1] [location=complement(5423501..5425840)] [gbkey=CDS]
MGPNKTRIRRLPLLPLRGLLVYPSMVLHLDVGREKSVKALEKAMIEDSMILLCSQAEVNIEEPAKEDIYRVGTISKVRQM
LKLPNGTIRVLVEGVVRAEIAEYIANDEFYEVNAKELPEEEVNDAEIDALMRTVLNQFEHYITLSKKVTPETLAAVSDID
EPGRLADVICSHLSLKIKDKQDILETVDVRSRLEKLLNILNNEREVLELERKISQRVKKQMEKTQKEYYLREQMKAIQKE
LGDKEGRAGEIEELRSQLAASEVPENVQEKIEKEIDRLEKMPSTSAEGSVIRNYIDWLLALPWTKKTEDDLDIHKAEEIL
NADHYGLEKPKERVLEYLAVQKLVKKLKGPILCLSGPPGVGKTSIARSIAKSMNREFVRISLGGVRDEAEIRGHRRTYVG
AMPGRIIQAMKTAGSVNPVFLLDEIDKMAMDFRGDPAAALLEVLDPEQNNTFSDHFIEVPFDLSNVMFITTANALHNIPR
PLLDRMEVLYLPGYTEIEKLEIADSYLLPKQKRDHGLEDGQMVMNRDALLQVIREYTREAGVRNLEQQLAAICRKTAKKI
VSEPGRTVEVTVDTVKDDLGPPKFRYNMADEQDQIGAVTGLAWTEVGGDTLVIEVTVMPGSGKLTLTGKLGDVMKESAQA
AFSYTRSRAEQLKIAPDFHEKNDIHIHIPEGAIPKDGPSAGITMGTALISALTNIPVSSQVAMTGEITLRGRVLPIGGLK
EKALAAHRAGIRTILLPKDNEKDIRDVPESVRNDLTFIPVAHMDQVLEHALVKEVDLKA
>lcl|NZ_CP026520.1_prot_WP_174446821.1_4880 [gene=lonB] [locus_tag=PC41400_RS24580] [protein=ATP-dependent protease LonB] [protein_id=WP_174446821.1] [location=complement(5425990..5427714)] [gbkey=CDS]
MSFSMILMLIQVFFAVVIGMYFWNLLRNQQTNRSAIDRESKKELEKLRKLRSVSLTKPLAEKTRPSSMQDIVGQKEGLKA
LKAALCGPNPQHVLIYGPPGVGKTAAARVVLEEAKKNPQSPFRNEAKFTEIDATTARFDERGIADPLIGSVHDPIYQGAG
AMGVAGIPQPKPGAVTKAHGGILFIDEIGELHPIQMNKLLKVLEDRKVFLESAYYNSEDNNIPNFIHDIFQNGLPADFRL
VGATTRTPNEIPPAIRSRCMEIFFRPLLPEEIGEIAANALAKIGFPANESAIEVVKRYATNGREAVNVIQLAAGLALTDK
REEVTAADIEWVVNSSQIQPRPERKIPSEPQIGFVNGLAVYGPNLGTLLELEVIAIPAARPGSGSFNITGVVDEEEMGGG
SRTLRRKSMARGSVENVLTVLKRMGLRTSDYDLHINFPGGTPIDGPSAGIAMATAIASALNQIPVDNKLAMTGEMSIHGK
VKPVGGVVAKVEAAFQAGATRVFIPKENWQELFADLPGVEVIAVESVEEVLSLALGLQIASNVIPMPAAGEQMLPGTMPF
LHAGPTDGSESAKS
>lcl|NZ_CP026520.1_prot_WP_042230543.1_4881 [locus_tag=PC41400_RS24585] [protein=N-acetylmuramoyl-L-alanine amidase] [protein_id=WP_042230543.1] [location=5427881..5428804] [gbkey=CDS]
MFTKRLLTGSCLAACITGLSLFAAACGTPNRLAETKALTADTASQASKLAADAKKSLTTGKAHAESLPDIMKPAMTPDSP
LNAQPPKSASKTNSHINPPDVLIDVGHGGIDGGAVHGDLLEKDINLAVAKQTYQLLTTAGYRVVLNRDKDYALSDDNKWL
NNRSRHIRDLAQRMQMANDLKPKLLISLHVNSSFSRSKKGPYVLHQNSAGSLFLATQIQERLNAISGTRHLPILGKTYYL
LKRAQSPSVIVEIGFITNAEDRARMTSAEGQQKIAAAIRDAVAAYITHGDLGPGPGGSQNKDSSPVK
>lcl|NZ_CP026520.1_prot_WP_042230707.1_4882 [gene=ispG] [locus_tag=PC41400_RS24590] [protein=flavodoxin-dependent (E)-4-hydroxy-3-methylbut-2-enyl-diphosphate synthase] [protein_id=WP_042230707.1] [location=complement(5428989..5430137)] [gbkey=CDS]
MFHRKDTRPVRVGDLTIGGSDEVIIQSMCTTKTADVKATVAEIHRLEEAGCQLVRVTVNNMEAADAVKEIKKQINIPLVS
DIHFDHRLALAAIENGIDKVRINPGNIGRRSKVEAVVKACKERGIPIRIGVNAGSLENHLLEKYGYPTPEAMVESALFHI
GILEELDFHDIIVSLKASDVPMAIAAYTQAAQAFNYPLHLGITEAGTLFSGTVKSAAGLGALLSMGIGSTMRISLSADPV
EEVKVARELLKTFGLITNAATLISCPTCGRLDIDLFSIANEVEDYIANLKVPIKVSVLGCAVNGPGEAREADIGIAGARG
EGLLFRYGEMVRKVPEAELLNELKKEIDIIVHEFNRTGEIPGRRQHQERIELAQKQAQAQHS
>lcl|NZ_CP026520.1_prot_WP_042230544.1_4883 [gene=clpX] [locus_tag=PC41400_RS24595] [protein=ATP-dependent protease ATP-binding subunit ClpX] [protein_id=WP_042230544.1] [location=complement(5430285..5431550)] [gbkey=CDS]
MFKFNDEKGQLKCSFCGKSQDQVRKLVAGPGVYICDECIELCTEIVEEELGHEEELDLKDIPKPKDICGILDQYVIGQDQ
AKKSLSVAVYNHYKRINSQSKLEDVELQKSNIILVGPTGSGKTLLAQTLAKILNVPFAIADATSLTEAGYVGEDVENILL
KLIQAADYDVEKAERGIIYIDEIDKVARKSENPSITRDVSGEGVQQALLKILEGTVASVPPQGGRKHPHQEFIQIDTTNI
LFICGGAFDGLEQIIKRRIGKKVIGFSSDGLKTDLKPGEYLSMVLPEDLLKFGLIPEFVGRLPVVSTLEPLDEAALVRIL
TEPKNALVKQYQKMLDMDNVKLEFDAESLDAIAKEAIKRNTGARGLRAIIESIMLDVMFDVPSRTDVTNCTVTKDTVQDK
LTPILSSQEGQDVNKKKEESA
>lcl|NZ_CP026520.1_prot_WP_009674201.1_4884 [gene=clpP] [locus_tag=PC41400_RS24600] [protein=ATP-dependent Clp endopeptidase proteolytic subunit ClpP] [protein_id=WP_009674201.1] [location=complement(5431563..5432156)] [gbkey=CDS]
MSLIPMVVEQEARGERSYDIYSRLLKDRIIFIGSAIDDDVANLVIAQLLFLQSQDAEKDIHVYINSPGGSVTAGLGIYDT
MQHIKPDVSTICVGMAASMGSLLLTAGAPGKRFALPNSEVMIHQPLGGVRGQASDIKIHADWIIKTKQKLNNIYVKHTGQ
PYDKIDRDTDRDNFMSAEDAKAYGLIDEVITRSDLKS
>lcl|NZ_CP026520.1_prot_WP_042230546.1_4885 [gene=tig] [locus_tag=PC41400_RS24605] [protein=trigger factor] [protein_id=WP_042230546.1] [location=complement(5432359..5433723)] [gbkey=CDS]
MKASWEKIEKNTAVLDIEVEAEQVADALDRAFKKVVAKVNVPGFRKGKVPRSIFEKRFGIESLYQDALDIILPEAYMQAV
QETGIEPVDRPEVDVEQFGKGQALKFKAKVAVKPEVELGDYKGIAIPAADSAVTPKEIDEELERLQQRHAELVVVEEGAA
ANGDITIIDFEGFVNGEAFEGGKAEKYSLELGSNSFIPGFEEQIVGMTKDEEKDIEVTFPEEYHSEQLKGQAATFKVKLH
DIKRKNLPELDDEFAKDVSEFDTLEEYKQDIAKNLEEKKKQDAEVQRETAVVEKAAANANVEIPASMIETELDQMVGEFE
RRLRMQGMTLDLYFQFSGQNVDVLKSQMREDADKRVRNNLVLEAIAIAEKIEATDEDVNAELDRLAEQYQKSVEELRSIF
EANGSLEGIKQDLAVRKTVDFLLANSTEEKEKPAKKEPKAKKAAKEEAKEEAAE
>lcl|NZ_CP026520.1_prot_WP_042230547.1_4886 [locus_tag=PC41400_RS24610] [protein=hypothetical protein] [protein_id=WP_042230547.1] [location=complement(5433878..5434792)] [gbkey=CDS]
MTHSYDNVVLFPKTAASYEKQMTQMLEQERYEEAVRLLRYLLQFARDDAEKQAQWRALLQWLETMFPETLYDDAVMQEGG
EEEEITEEDLKRQSVAGKSAEGRQYVGRLLDMLEDGTDMERQMIALEQLAFAEDDRIDEFVKEWLHTAESHPFVQFRALQ
MLKQRGGKGPVVLARAQNKMVLRIEDTPLNLEEFPDRIREMVYRVQQISESEQPDFGFFAEQTWLEFLKTVYGTELYTSM
SSCREEEVDVWASALHTVLHESLFGSADREHMREQYGITASMLQAWGEAYACLQSFIRALYTGG
>lcl|NZ_CP026520.1_prot_WP_042230548.1_4887 [locus_tag=PC41400_RS24615] [protein=PIG-L deacetylase family protein] [protein_id=WP_042230548.1] [location=complement(5435039..5435776)] [gbkey=CDS]
MRKTLLFVFAHPDDESFACGGTMAKYAESGHAVHLACATSGCKGKTGEYTFTCREEIAVYREGELRRAAAVLGVERVHFY
RYPDGDLAEVREDLLAARVAATILDLQPHLVVTFPPDGVTGHPDHIAISKAVLRAVKELDAEGGAAPDLYYASIPNYYDH
CGDCGPSECCPITARVNISEFRSRKGEALQAHKSQVYSVDRAYPGVMHGDYSVIGPYEYFTLVRRGGQSTGLNKCAGEIP
VVELV
>lcl|NZ_CP026520.1_prot_WP_042230549.1_4888 [locus_tag=PC41400_RS24620] [protein=BadF/BadG/BcrA/BcrD ATPase family protein] [protein_id=WP_042230549.1] [location=complement(5435801..5436757)] [gbkey=CDS]
MKGRSIPLLAVDGGGTKCLAVFTDRQGKIEGTGRGGSCNYQTTGKEAAVDELVRAMKIAKEELGESPDETLRVECAVFGM
AGLDTGYDRQIIEGLVREALSRAEIRADRVVVENDGLAALLGATDGKPGVLLIAGTGSIVYGVNGQGRSARAGGWGYRVG
DEGSAYWIGKQALTAILQTLDGRQGPTSLARRVLPYLDLRSEEELYNWVYSADYRVDTVAELSLLVSDSAREGDASSLRI
LEAAADELFYGGRAVIEKLGMTDTSFTLILQGGVLQNNEFVRSRLTKKLREFAPSCNLEEIKKEPIYGVIAQGLSLLP
>lcl|NZ_CP026520.1_prot_WP_042230550.1_4889 [gene=argH] [locus_tag=PC41400_RS24625] [protein=argininosuccinate lyase] [protein_id=WP_042230550.1] [location=complement(5436754..5438265)] [gbkey=CDS]
MRNREQILQEEGAAFPGLTYTEAVLKPAFNEAKASLLGPMMAINKAHLIMLKDQGLVTEEEASQIADAIRGLDLEGLRQA
EYTGQFEDLFFQVESDLLEAAGDIAGNLHLARSRNDMGISIYRMVLREKLQTTIRGALALKSQLLAFAEEHAETIMIGYT
HTQQAQPTTLAHYIMAVADSLDRDVRRLQAAYANCNRSSMGSAALTTSGFNISRERMQELLGFDELIYNSYDAIGGADYV
GEIATAVQLAAINLGRSSQDFLLWCTQEFGVLRVADPYVQISSIMPQKRNPVSFEHMRSLLSSCVGNTQTVLTMMHNTPF
GDIVDTEDDMQPYAWKALAVLNQMYRLLSCVIGTIHVDKDVLRKRTEGSFATVTELADTLVRTDNLSFRKAHHIVSGVVK
RAVSEGLAANEISLAMVNEIARQVIGRDLSITEEAFRSSLDPVHFVYIRTLPGGPAPDEIRSTIADRRSRQQTQEAWLRD
SRSKTEEALSHLDAALEGWSSGQ
>lcl|NZ_CP026520.1_prot_WP_042230551.1_4890 [locus_tag=PC41400_RS24630] [protein=creatininase family protein] [protein_id=WP_042230551.1] [location=complement(5438326..5439072)] [gbkey=CDS]
MKALDIHGRHAEDAFRNAAFAVVPLGSLEYHGPHSPLGTDFFLAQGFADLIDPGLNPLVYPAIPYTACPGKTGDYPGTIS
IDPAVFHRYLSDVLEGICRSGFRRIVLLNAHDGNIGASRTVAEEVTSRWKDASFLLINWWQLAAADAPEKLGFFQGTTGR
GHGGPYEMSAVKAFRPDLVHVSEEDAELTPPPPLSPMPYVLVEGTPEGWAGYTGHIRQTSLQTGDWIVGEVTANMNALIA
NWLGRPAP
>lcl|NZ_CP026520.1_prot_WP_042230552.1_4891 [locus_tag=PC41400_RS24635] [protein=PIG-L family deacetylase] [protein_id=WP_042230552.1] [location=complement(5439240..5441765)] [gbkey=CDS]
MKKWVAAAVSLTVVGTVALPAAPQTASADRGVVDLWKSIKPLTTIASAMNTGAHPDDEHSATLAYVSLGQGVDTSSVIAN
RGEGGQNEIGSELGNALGIIRTRELQEASKITNITLANLSVKLDDPIYDFGFSKSPDETLEKWGESVAYERLIRQIRQKR
PDVLIPAFLNEASTHGHHRAINVLTVRAFKDAGNPAIFPEQLKEGLAVWQPKKLYLPATDKEYDVALPVGDYNEIYGASY
SQLGEESRFMHKSQGMGRHVDEGPGNSYYKLAGTAGDNKTAVSKSSSLFDGISFTFEDLAKELDTKGKDNKIVKDLKTLQ
QDANEVVAAYPKFANVAREVHEMKADVKSAIENVEKSTLDASVKTDLLHRLQVKTDQLNKASMEAASLTVKVKPVSGELV
AGQTAKVAVTAHSGGSVDISDIAMKLNVPEGWTAKAAGAAEFKKLGYNQTVKTEFEVTVPAKSADFNPYALPLISADVTY
TAYGTQAAVHAVPDNAVAVLPATSLTLSPEAAVFNTLKPDEAVPVKVTARNYAPGASKTTVSLKAPEGWTVEPASLDVSF
AAKNETKAVEFTVKAPKGVKPGKVDISAVATEGQTDTSRTVQVISYPHIGTTYYVKPATLTIQAFDLNMPQNLKVGYVSS
GFDNIDAVLRQVGVNVTNLDAKAIQFGDLSQYDTIVLGIRAYAFRPELIPSNQRLLDYAKNGGNLVVQYHKPEDKWTPDL
APYPIKIGQPLIQWRVTDENSKVTMLAPDHKIFNTPNKITDADWQNWIQDRSAYNPAEWGKEYTELISNGDPGEKEFTGT
FLSANYGKGTYTYSSLVWYREMPSLVPGSIRMFVNMISLKQ
>lcl|NZ_CP026520.1_prot_WP_042230554.1_4892 [locus_tag=PC41400_RS24640] [protein=iron ABC transporter permease] [protein_id=WP_042230554.1] [location=complement(5441928..5443673)] [gbkey=CDS]
MQTNKPQAKQQPAIRPNSRWTTITTSPYFVYVLIAPLFLILLAYVIYPFFQTFLQSVKLEGAFSFKNYARFFSLEHTSNL
EALWNSVYISVLSVITCGIVGVAMAFLLERYEFPGRKILSVLAIVPMALPPLIGVLSFSFLYGDSGIIPRALQKLFALDH
APLALKGIWGVIVVHTFTMYTYFYMTASTAIRGLDPSLEEAAANLGAGRLLIWRRVILPMLTPAIIAASLLVFMISMASY
TAPLVFGIDRTLTMQIYLSRTNGDLDMAATQSTLLSIVSILFLIMMRWYQNARNYQNMSKGVSVHRTEVKSKAGKYTAMV
LSFIATLVLMLPILVIVLLSFSEDSTWTVQVLPPEYTLQHFKDLFTDPKTWRPIANSFKMSFVATIGNLLFGVAAAYAIV
RMKFKGKTLLDILIMLPWALPGTVVAVNLITAFSEPNAFSFNQVLIGSFWILPLAYFVRHLPLIFRSTSSTLMQMDSSVE
EAARNLGASWWYSFRRVVFPMALSGILAGTLLAMVEGLGEFVASILLYTNNTTPMSVEIFQRMYAFEFGTACAYGVLQIV
MIILVLFVSRKLTGGNAGSAI
>lcl|NZ_CP026520.1_prot_WP_042230556.1_4893 [locus_tag=PC41400_RS24645] [protein=ABC transporter ATP-binding protein] [protein_id=WP_042230556.1] [location=complement(5443687..5444769)] [gbkey=CDS]
MIEVKLDKVSKQFNGARGVENVDIHIQPGEFFTFLGPSGCGKTTTLRMIAGFYYPTEGRIMFGENNVTYTPPNKRNTGMV
FQNYALFPHMTVFENIAFGLQVRKQSKAEIMDKVTRAQKLVHLDGYGDRRIDQLSGGQQQRVALARALVIEPNILLLDEP
LSNLDAKLREETRLEIKRLQLELGITTIYVTHDQAEAMSMSDRIMVMKDGIVQQIGTPHDIYNRPVNRFVASFIGESNLW
EGTVERMEGDSVYVRTSSDVLIAGLAANASPECRLALGSKVTVSVRPESIREAAASERGLENILAGKVLMSEFTGVSVNY
VAQVGTLPIKAMFIGPGQRMRHRDEDIAFAVPKESVYFVE
>lcl|NZ_CP026520.1_prot_WP_174446822.1_4894 [locus_tag=PC41400_RS24650] [protein=extracellular solute-binding protein] [protein_id=WP_174446822.1] [location=complement(5444913..5445989)] [gbkey=CDS]
MTKWTSLTLIAAMAVGLSACGGGDDGAKKGAEGAGGGAASGSQKLVIYTARDKNVVDEILPRFKEKNPGIDVEVMTMGAQ
QVLERIRGEKSNPQADFWWGGTQSSFITGAEEGLLEPYKPTFADNVAAENKDAQDRWYGEMLLPEVIMYNSEALKKEDAP
KDWDELLDAKYKDKLLIRGVMASGTMRTIYSSMIYRQDPANPQKGYDWLKKLDANTKEYTQDPTNLYLKMARQEGTLSLW
NLQDILIQKEMKKQPFDFIYPASGAPILVDGVGIVKGAKNMDAAKKFYEFLFDKEELKKKAEKLFQIPTRTDINKDEFPA
WLKGIETKPLQLDWKVLASKEKEWMQYWDENIKGKGKK
>lcl|NZ_CP026520.1_prot_WP_042230558.1_4895 [locus_tag=PC41400_RS24655] [protein=ROK family transcriptional regulator] [protein_id=WP_042230558.1] [location=complement(5446033..5447232)] [gbkey=CDS]
MIGKGKTGNTKLVKQMNREGILYQLRQHPRLSRAELSVRTKLSRPCVSSLVDEMIHEGLIHEVGTGESKGGRKPILLEYN
VQAYTVVGAVFEGSELEMAIADMKGELLHHVHTRLEQPVDGDSALEALKTNLDTLLEKSGFGRDRVIGIGVGLPGITQKR
SGTVSYAPSTGWMNLPVGKEIEERLGIPVILDNDVNMMTMGEFYGGAGSGVSSLVYMYVGTGIGAGIIIDKQFYRGSKEA
AGEIGYMLIGGDRLRRQGEFGVFEKNYSVTGICDKAKAAGLAVDPGRSILRQLVEQADQGSRQAGTLLEDVYVHWASGIA
NTVSVLDPELLILSGEMVHLSESGLTRIRELLAQWVPVLPDIQLAGLRERAGLLGAVHSVLEAFPSAQPARSSGSAGAD
>lcl|NZ_CP026520.1_prot_WP_255246842.1_4896 [locus_tag=PC41400_RS30640] [protein=hypothetical protein] [protein_id=WP_255246842.1] [location=complement(5447494..5447622)] [gbkey=CDS]
MAKVSKATKGKKYEVNAEFAEEMEAKNVKHSAQNNLSHGTAK
>lcl|NZ_CP026520.1_prot_WP_084706608.1_4897 [locus_tag=PC41400_RS24660] [protein=DUF4023 domain-containing protein] [protein_id=WP_084706608.1] [location=complement(5447711..5447824)] [gbkey=CDS]
MDSHFISKLHDTQRKDERNRRTQGKNRPDKRLPNKQH
>lcl|NZ_CP026520.1_prot_WP_042230560.1_4898 [locus_tag=PC41400_RS24665] [protein=hypothetical protein] [protein_id=WP_042230560.1] [location=complement(5447991..5448344)] [gbkey=CDS]
MKRKYARAQIDYLAAKEQFLAASQQADKKLNDMRQDGAEIGQKEMEDAIERSGFHKAFNELRQAENRLIEWSHFTIKNEG
EFKENKAAFEKLYNEVKTKPEARAVLVDMAMRLEAEA
>lcl|NZ_CP026520.1_prot_WP_042230562.1_4899 [gene=asnB] [locus_tag=PC41400_RS24670] [protein=asparagine synthase (glutamine-hydrolyzing)] [protein_id=WP_042230562.1] [location=5448585..5450429] [gbkey=CDS]
MCGITGWIDWRKDLTTSSSILENMTETLVHRGPDASGTWISGHCALGHRRLSVMDPLNGAQPMIRKDGDRTYSIVYNGEL
YNAPELRSELEHRGHRFATSCDTEVLLAAFVEWDRSCLERLNGIFAFAAWNEQEQTLFLARDRLGVKPLFYSHTDGAFLF
GSEPKAILAHPDIKTEVGAEGLAEIFAVGPARTPGHGIYRNLAELKPGHFMVYDRSGLTIRPYWKLENRPHTDTVEETAL
RIRELLADTARRQLVSDVPVCTLLSGGLDSSALTTLASHHYRDSGGGVLNTYSVDYADNDKNFKENAFQPNSDAPWINRM
TEYLGTEHHYIEFDTPELVESLRTVVLARDTPGMADIDGSLYLFCREIKKGATVALSGEAADEIFGGYPWFHREDALNAN
TFPWSLSLPQRAGILSRDFADWIKPEAYVRGRYEQALAEVPRQAGETREQNRMREMSYLNITRFMPTLLDRKDRMSMAVG
LEVRVPFCDHRLVEYVWNIPWDIKTAGDREKGILRKALVDVLPEDVLARKKSPYPKTHNPNYATAVKQQALEILNDSSSP
LLPFVDAKTLRAMAAPGAESTEFPWFGQLMSGPQMFAYLIQVDTWLREYKVSIR
>lcl|NZ_CP026520.1_prot_WP_084706609.1_4900 [locus_tag=PC41400_RS24675] [protein=response regulator transcription factor] [protein_id=WP_084706609.1] [location=complement(5450769..5451371)] [gbkey=CDS]
MKVIVAEDQGMLRGALSALLDLEDDIEVIGQAADGAEALEMIRTLRPDLCIMDIEMPKMSGLDVAEALQKSKDPCGVVML
TTFSRSGYFQRAMRAGARGFLLKDSPVGELAAALRDVQAGRRAVSPELALTFWEAENPLTEREREVLKLAREGLSSGEIA
SRLFLSGGTVRNYLSEAIQKLEAKNRIDAIAAAEKNGWLD
>lcl|NZ_CP026520.1_prot_WP_042230564.1_4901 [locus_tag=PC41400_RS24680] [protein=sensor histidine kinase] [protein_id=WP_042230564.1] [location=complement(5451368..5452519)] [gbkey=CDS]
MFRRQPARPREGPPFIFTIIWLVYLIFPVVSVLQLPFPEKILGSGLLIVFVAAYLTSFHRAAGRYWVILFQTVVIGFLSI
RYNENFIYLAFYPSPVIGMLRSVRQMAAAIAGLFLMFVAVGWHYKLMNDVDLFLQLLPALVIILIMPVAMRIGRRSKELR
GKLQLANEEIARLSKIEERQRISRDLHDTLGHTLSLITLKGELAEKMIPKNPEKAVREVQDIQATSRAALKQVRELVSDM
NMVTIREEFDRAKQILAAGGIVLELKAGEGAEDVAPPLIGNILGMCLREAVTNVVKHSKAKLCIVEWASDEEGYRLIVSD
NGVGFDRDERQPDLGTNGLRGMKERLKLVDGHCTFESAGGRGTRVVFTIPKVTKNGEKEAGGE
>lcl|NZ_CP026520.1_prot_WP_042230566.1_4902 [locus_tag=PC41400_RS24685] [protein=ABC transporter permease] [protein_id=WP_042230566.1] [location=complement(5452520..5453275)] [gbkey=CDS]
MNATILKATMAQCRAELLRTFRNKRFIFFSIVMPVAFYFLFTSTMGSDTKIGSVDWTSYYLMSMTVYGVIGASLTSFAQR
ISRERTQGWVRLLSITPLPSWSYMVSKVAAQGVINLFIILMMFIIGGLGKQVDLPVSVWIESFLWIWVGGFSFMALGTLL
GSMRNPDVVQVLTMILYMGLSVLGGLWFPTASMSGTLQTIAKFTPTYRLGQGAWNLIGGGSIDWTGVLILAVYVVVFVLL
SSYIIKRQEAV
>lcl|NZ_CP026520.1_prot_WP_042230568.1_4903 [locus_tag=PC41400_RS24690] [protein=ABC transporter ATP-binding protein] [protein_id=WP_042230568.1] [location=complement(5453294..5454190)] [gbkey=CDS]
MTTAIELRQVTKQFKGKKAVDGLTLTVEQGTVLALLGPNGAGKTTTISMILGLQEPTSGTVKLLGGHPKDQRVRDRIGAM
LQDVSVIDNLKVAETIDLFRHYYANPLSLEQLLLISGLEAERNKMASALSGGQQRRLGFAMAAAGDPEIIFLDEPTVGMD
VSSRQLFWDTIRAMAARGKTVVLTTHYLEEADQLADRVVVINNGKLIADGTPSEIKAATTGRVISFTAGAGVTHELLRAM
PGVSDIKWSGRRVEMFGPDTDRLLTALIDKRVDMKDIEVRSGGLEDAFQTLIQKTNAG
>lcl|NZ_CP026520.1_prot_WP_042230569.1_4904 [locus_tag=PC41400_RS24695] [protein=non-canonical purine NTP pyrophosphatase] [protein_id=WP_042230569.1] [location=complement(5454453..5455133)] [gbkey=CDS]
MPDRSASGVVVIATRNKGKVREFAHYFARFGKEVRSLDDFDDMPDIVEDGATFAANARIKASAIAGRLNLPVLADDSGLC
VDALGGDPGVYSARYAGEHAADAANNAKLLRELGAVSAAAARAPGPPDGPEAADAAAPQVLSTARFVCALALVDPLRGDT
LEAEGTCEGVIIAEPRGTDGFGYDPLFYLPERGLTMAELPLEEKNAISHRAAALRQFFERHSGRLT
>lcl|NZ_CP026520.1_prot_WP_042230571.1_4905 [gene=rph] [locus_tag=PC41400_RS24700] [protein=ribonuclease PH] [protein_id=WP_042230571.1] [location=complement(5455126..5455875)] [gbkey=CDS]
MRADGRQNDQLRPMKITPHYIKHAEGSVLIEVGETKVICTATVEERVPPFMKGQGKGWVTAEYSMLPRATQVRNQREAAR
GKLGGRTMEIQRLIGRALRSVVNLEALGERSVTLDCDVIQADGGTRTTSITGAFVAMALAMEKLVANGKLSKLPITDYLA
SVSVGIVNGKPLLDLMYTEDSQAHVDMNVVMTGTGKFVEVQGTGEEAPFSREELNTLLALAEGGINTMVEEQKKALGRVS
GLIGAGARA
>lcl|NZ_CP026520.1_prot_WP_042230573.1_4906 [locus_tag=PC41400_RS24705] [protein=GerMN domain-containing protein] [protein_id=WP_042230573.1] [location=complement(5456066..5457121)] [gbkey=CDS]
MRKVRWLQAAAMTGAIAILGTGCGMWGSESKPIDAPPPGTETAMETGTGTGGTGAEAGAKTSQTAKMTVYAKDPKGFVVP
VSLDLPKTQSVAKTTLEYMVAKGPGESQLPAGFASLLPEGTTLTVDIKDKLATVNFNQAFKKYDAKEERKIMEGITWAMT
GFSSVEQVKIKINGQELKEMPVAHTPLDEPLSRKIGINLEKQEGVDYGQSTPVTLYFTNATAQNGKYYVPVTRLVKRTDD
VAKTVMDELLKGPNTSKGLQPVLNASAKVLEVKKTSDTVLTVNFSDKLLDENKKAPADGLQSVILSLTENTNAAKVQITV
NGSPKVETSDSQTYSQPVSRPLHLNPNPEKL
>lcl|NZ_CP026520.1_prot_WP_042230575.1_4907 [locus_tag=PC41400_RS24710] [protein=phosphatidylglycerophosphatase A] [protein_id=WP_042230575.1] [location=5457430..5457957] [gbkey=CDS]
MKYSLDSEKVASATYELLEKRGVQTSQIAELTYFLQKDYFESLTLEQCLASVESVLAKREVQNAVLTGIQLDMLAEEGKL
ISPLQEMIQHDEGLYGCDEILALSIVNVYGSIGFTNFGYIDKLKPGILKKLNDKSTYEKHTFLDDIVGAIAAAASSRIAH
RKQSEAETKGDAPGS
>lcl|NZ_CP026520.1_prot_WP_042230577.1_4908 [locus_tag=PC41400_RS24715] [protein=hypothetical protein] [protein_id=WP_042230577.1] [location=complement(5458260..5458664)] [gbkey=CDS]
MTGRKTPKQRLIAAGFGLTSLVVIGFTLKKYASGEPVGTSELTSIAAALAAMFGALTWGNKEDGEGITQEEEMGRHITFV
SAKAAYFIMLAASLIVCIVEKLAFGRDNMAVLILLGLGIVLLPAAEYIVARKYR
>lcl|NZ_CP026520.1_prot_WP_042230579.1_4909 [locus_tag=PC41400_RS24720] [protein=helix-turn-helix transcriptional regulator] [protein_id=WP_042230579.1] [location=complement(5458693..5458893)] [gbkey=CDS]
MENRIHELRKRIRLSQEDLAKACGVSRQTINAIENNKYDPSLQLAFHLARTLGVTVDELFQADKDT
>lcl|NZ_CP026520.1_prot_WP_042230581.1_4910 [locus_tag=PC41400_RS24725] [protein=MBL fold metallo-hydrolase] [protein_id=WP_042230581.1] [location=complement(5459033..5460025)] [gbkey=CDS]
MSHSTTVIALREGLWQIKVPLPFPLRWVNSYLLREPDGSLTVIDPGLRTEAAEALWNEALEGLGFGYDRISQVVLTHHHP
DHYGMAGWFQERTGAPVYLSETGIGQVRLLWGDGQPMTSLLTGLFVRHGMDAALCGELAKHMDSFVPLVSPQPQLTALRA
GDRVRLGGTAYEAIETPGHAAGHLCFYDAQTREIFCGDHVLPQISPNVSFMPGVEDDPLGAYMASLRDVAPLPVEMAYPG
HRSPFAGFGRRAEELVRHHEERLELMADKLHEPMTAYELCRDTFGDRLTLHQLRFAMAETIAHTVRLAVEGRVTQSEREG
LIVYEARDEG
>lcl|NZ_CP026520.1_prot_WP_042230582.1_4911 [locus_tag=PC41400_RS24730] [protein=class I SAM-dependent methyltransferase] [protein_id=WP_042230582.1] [location=complement(5460022..5460777)] [gbkey=CDS]
MKVKSAWYRESFGKDYLLVYKHRNFEGAYAEVKRMMNWLHLPEGARVLDLCCGMGRHSLALADFGYTVTGIDLSDTLLGE
ARKQDPEERVTWLHGDMRRVPSEESFHAVVNLFTSFGYFEQEEENRLVLSEIARLLQPDGKFIIDYLNPAYVKTNLVPHS
SREENGLFIEETRSIENGYVRKQICITDNQDGPRHYSEQVRLYSCEAFVDMLYTAGLRVERIYGSYDGKDYDRESSPRMI
FVGGREGESLQ
>lcl|NZ_CP026520.1_prot_WP_042230711.1_4912 [gene=corA] [locus_tag=PC41400_RS24735] [protein=magnesium/cobalt transporter CorA] [protein_id=WP_042230711.1] [location=complement(5460833..5461783)] [gbkey=CDS]
MIRTVAVTRSGELVLDLPLQELGGDSIDWYWVDFNNPTHEEAMLLDTHFHFHPLAVEDCFHLLQRPKMDHYEDVHFFVLH
AIHPQTLEADEVDLFVGPRFLVSFHFTELKEVDDAWQKLQAEPDNWKRGSSFGAYLVMDMLVDEYFPCIHMLEDEMNDFD
SDQTGTSIHALMRGVFDIRSKLAHLRRTIIPMRDLLYRVVNSDKLPNEREQKIYYADIYDHLLKLAEMIDSIRDTTADMR
ENFISLNSYRMNTIMKTLTVITTIFMPLTFLAGIYGMNFDNMPELHWHYGYFIVLLIMFGIGFAMYVWFKRKGWFD
>lcl|NZ_CP026520.1_prot_WP_042230713.1_4913 [locus_tag=PC41400_RS24740] [protein=LysR family transcriptional regulator] [protein_id=WP_042230713.1] [location=complement(5461901..5462794)] [gbkey=CDS]
MDVRHLQYFLETAKRRSFTKAAEALHVTQPTISKMIRILEEELGVELFDRTARQIVLTDAGEAVMIQAQQIVRSLENLSV
DLADVTSMRKGQIRIGLPPMTGAPFFPELIAAFHKLYPRIGIRLIEEGARTIEREVESGSLDLGVAVLPVDAERFDMFTF
VKEELRLVVHHGHRLARRQTVRLVELSGEKFLMFREGFTLHESIPAACIQAGFKPDVIYESSQWDFLSGMAEAGLGVALL
PETICRKLSPDKVATITLVSPRIPWHLVMIWRKEGYLSYAAREWLRFSKERLSLPNL
>lcl|NZ_CP026520.1_prot_WP_042230584.1_4914 [locus_tag=PC41400_RS24745] [protein=CidA/LrgA family holin-like protein] [protein_id=WP_042230584.1] [location=5462927..5463403] [gbkey=CDS]
MNKQPAPPLSGPHKKQNFVWPPDRFAPVHVRRALVTCGQIVLLIAFAYLGRLVSERLHLPVPGSIVGILIVFAALHFGWV
RLEWVEKGADFLLSKLLLFFVPPAVGILEYKEMFSVSGSRYLLIIIISTFFVMASTGLVAEYLGRRRAGKDKKEEASL
>lcl|NZ_CP026520.1_prot_WP_042230585.1_4915 [locus_tag=PC41400_RS24750] [protein=LrgB family protein] [protein_id=WP_042230585.1] [location=5463400..5464083] [gbkey=CDS]
MTGTGLLYLLITIVVFWAANRLYRKTGKAYLSPLVVAPVLLIAGLFFTGTPVEAYNAGTKPLTDLLQPATVAFAIPLYRQ
FKLLRKHAIVLVAGVLCGSVVAVLTSVLPAGWIHASPQMIESVAPRSITTPIAMEVSRSLGGIPSVTAVFVVVTGICGSV
IGPLVIRSLRIRSGISRGLLLGMGAHGAGTSKAHELGSEEGAIASVAMILAALFSLGLIQPIMAPFL
>lcl|NZ_CP026520.1_prot_WP_053228721.1_4916 [locus_tag=PC41400_RS24755] [protein=S-layer homology domain-containing protein] [protein_id=WP_053228721.1] [location=complement(5464401..5466371)] [gbkey=CDS]
MRHLHKLLSLMMIVCLLQVGPLVHAETVLQNQAVNGGTINQNIVFIDVNNHWANDAIYDMAKKGIISGYEDSSFKPEKSI
TREEFAKLIALTFSINLQTPAEPTFADLNPDKWSFGYVESTKEFLTGYYPPKGKPFFSPDTKATREDVAVALVKTMGYTT
DDLRSSSTLGRNFSDVDEISYNLRDYVAIATEKQLIGGYEDSTFRPQKPITRAEVATLLFRVIKSSAQDQADGPSLEVTV
PEKTESGTFYVSGKTHRDAKVQINDKAVSTDEGSFKEGYRLEKEGTYEIRITAKLPSGKTTTVTKKVTYQVNGPDLKVDN
TPDTVTKGSLTISGKVKKDTNYSYPTVYLNDEKMYVSSWDGTFSKEVKLAEGENTFTVKAENEAGKETVVTKKVTFSSGG
PELRVNSIPDSSQVASIKVSGTVKDANDSYPILYINDKKVYMYGGSFSESVRLEEGENTLTFKAENQNGKVTVVTRKVTF
NVNGPTLKLNATPEKTASKTLTVSGYVTDKNDSMPTLKINGKKVYYTSYASGSFSEEITLSEGENTLIFEATNSTGRSTT
LTRTVTLETGAPIIQVSYHPGSTTLSYMTISGTVTDKNDSYPVVYLNDRKIYSSSYDGSFSEWLDLKKGANPYTIVVSNK
YGKTATVTGNVYRVDR
>lcl|NZ_CP026520.1_prot_WP_042230590.1_4917 [locus_tag=PC41400_RS24760] [protein=stalk domain-containing protein] [protein_id=WP_042230590.1] [location=complement(5466650..5467375)] [gbkey=CDS]
MRKRWVAAGMTAALLVGVTAGAYATSNLKEIKAFLNPDLQVDLNGYTFALKDEQGSDLVPITYEGNTYLPLRDLAQRLDL
SVTWDGTNNKIFLKNKPYMEFSAEDEAMLEKYGYGFRVPRWLENVVVSVIPEENVKTSIEEGRLDKRAQALTLIHYMVQD
ENLHRLHDVAARIEVYKKEDWPEVKKTAAGAEVLGENDRYVYIAKKSAENPYEPGTLDAEGFGDLRDWLDLHDYYFTLSG
K
>lcl|NZ_CP026520.1_prot_WP_009672290.1_4918 [locus_tag=PC41400_RS24765] [protein=response regulator transcription factor] [protein_id=WP_009672290.1] [location=5467610..5467834] [gbkey=CDS]
MKGSDHKNKFLLTNREREVFELLVQDKTTKDIAQQLFISEKTVRNHISNVMQKLNVKGRSQAVVELIKLGELKI
>lcl|NZ_CP026520.1_prot_WP_042230592.1_4919 [locus_tag=PC41400_RS24770] [protein=GrpB family protein] [protein_id=WP_042230592.1] [location=5468272..5468823] [gbkey=CDS]
MSAQQHYLVNIVPYSSDWSVEFEKEKERLLGILAPHIAAIEHTGSTSIPNQGAKPIIDISAAVHTLLDKQIYENLLSNSG
YRYIGAFKTGRHFFVKEHAGARTHHLHIFPLEGFYERNELLFRDYLRAHPDFVIEYGELKRTLAEQYPSDPEGYTRAKTD
FIQRVVDLARTERGLPLQNVWVD
>lcl|NZ_CP026520.1_prot_WP_042230595.1_4920 [locus_tag=PC41400_RS24775] [protein=hypothetical protein] [protein_id=WP_042230595.1] [location=complement(5469274..5470593)] [gbkey=CDS]
MQTNGIYASSTPSTIVAKNNQLATKVLVPASNNASDAVEMYTPSVIDEGQQELRALGGQWIVGREFTSVQGTDYSKNFLD
DVSGFLVKSAYERDLLLQRYTGDELSQHLEQYDKQVQLKIEDIAQTTASTLSQYLNFENDSIFGGSKETQGEFLSTYKPI
FDKDELTDRIKNMLQEQSRLLGKLQEEQGDQWKQIMEQGGKGITDFRDLLKEEASKASVGESTSFQDISYRDVAGVFDAL
VHTTAKAMNTFTFDTEWMANNLGTARLKLDLYLQQSPMSSAIKEQLTQSVDKAMSTKMGIAEYDDYKFSLISTPSFRKQF
ASHQEMSLEIKRKYSAHLQDLNAKKSVGTIMWESMKGLLNQKPEDLLSGLMNAFNSLHHSLISAPSMQGDVGPKAKLEFA
NQVRDANLEDWNSFISSLNMGQQQKSALTIPKNSIDIKV
>lcl|NZ_CP026520.1_prot_WP_042230599.1_4921 [locus_tag=PC41400_RS24780] [protein=helix-turn-helix domain-containing protein] [protein_id=WP_042230599.1] [location=complement(5471112..5471975)] [gbkey=CDS]
MNTNELRNKRHDEQRIFFKVDVLAVRGGIIAKIGADGFATLATIASFMDEDGVCYPTQETLAEMMGVTRFTVIKRIRTLL
ETKLDDGTPVLERAWRNRTGGGVGKQSVYKVNMACGVSMGRDSKDVKSIDVKESHVSEIHVEEKHALMLKNGTNNVYNFN
VLDVVAASHKEEPVFKKNQEEEPIKEERKDKLSYLSIPSENANDNNIHLPQVVKEDVEHKSFNWGEEWEKDFGDADRERR
RKSERPLTAKEKADIEEIQNSRTVTYSLSDEMFEAAFGVKNNTSTSC
>lcl|NZ_CP026520.1_prot_PC41400_RS24785_4922 [locus_tag=PC41400_RS24785] [protein=DNA-binding response regulator] [pseudo=true] [partial=5'] [location=<5472178..5472267] [gbkey=CDS]
HITNVMQKLNVKGRSQAVVELIKLGELKI
>lcl|NZ_CP026520.1_prot_WP_042230601.1_4923 [locus_tag=PC41400_RS24790] [protein=hypothetical protein] [protein_id=WP_042230601.1] [location=5472386..5472742] [gbkey=CDS]
MRKIFMPALLLVILCAGYLFWSGTAQYTISPEGYPVPRNAVLKKTIPDSAGTIHVYTAPGIHQTDGLKNSSKRQIEKHGW
TYAGDLSAGATYMFKKSDGTLLNVSIYEGEFSICQLQK
>lcl|NZ_CP026520.1_prot_WP_042230604.1_4924 [locus_tag=PC41400_RS24800] [protein=hypothetical protein] [protein_id=WP_042230604.1] [location=complement(5473212..5473844)] [gbkey=CDS]
MVVFTIDIRKVYVYTHRLQKDGTFKEVEQNLSIQEGDAVFEDYDLAEQWMKNNPVVYVDDEKIYNESETYSFGSPAFNIR
QHRTSLPQNIFFTKKEVANVLENGDDEVNHVLVVDYDGIVHLLPECSNLRGYAVRFETFIAGKGYVGSTGNLRHLDETYR
ALLDAWVEHLHHGDEAYRDYDIGKITVEELQKEAEALVAKMRTLGSEKES
>lcl|NZ_CP026520.1_prot_WP_241688921.1_4925 [locus_tag=PC41400_RS24805] [protein=hypothetical protein] [protein_id=WP_241688921.1] [location=complement(5473947..5474129)] [gbkey=CDS]
MKGKHSKTPAYPVYVKFKDEPGVTYVYTHNGDKWIQLEAARDQYGNLPKYKHDENPNLIK
>lcl|NZ_CP026520.1_prot_WP_042230605.1_4926 [locus_tag=PC41400_RS24810] [protein=DUF3139 domain-containing protein] [protein_id=WP_042230605.1] [location=complement(5474304..5474648)] [gbkey=CDS]
MPCVRLKKSSLIIGIVILLLALSPFAYSRYTLYNLERGILAYLTETKHYEEKEILSIEAKISKTPRYPIYVKFADEPDVT
YLYIYRDEKYIQTPPTRDARGNLILHKHTEKETL
>lcl|NZ_CP026520.1_prot_WP_042230607.1_4927 [locus_tag=PC41400_RS24815] [protein=pyruvate, water dikinase regulatory protein] [protein_id=WP_042230607.1] [location=complement(5474843..5475661)] [gbkey=CDS]
MVQDAHEAQQTITICSDSVGETAETVVRATIRQFNGHNVKMRRIGHIKHEDEIREIMEEVHREGGFVAYTLVQPELREMM
KEEAIRLGVRAIDIMGPMMQAYIDTFDGAPSRKPGLMHELDDAYFRRVEAIEFAVKCDDGRDTTSMMKADIVLIGVSRTS
KTPLSIFLAHKGYKVANFPLVPEVKLPKELFEFPGSRIVALTMNAEKLFKIRIERLKTLGLPFGSQYATMNRINEELEHA
ISIAKRLRCPVIDVTDKAIEETAGLIIEFLQA
>lcl|NZ_CP026520.1_prot_WP_042230609.1_4928 [locus_tag=PC41400_RS24820] [protein=helix-turn-helix transcriptional regulator] [protein_id=WP_042230609.1] [location=complement(5475682..5476293)] [gbkey=CDS]
MEILELIHKQAPITGEQIAEQLGVSRPTIRSDLSLLVMLGHINAKPKVGYFPNQSFREEQAGGLKLKVKQVQSMPVIVRK
TATVGDAVVALFVDNVGGLVVADEEGMLLGVISPKDLLKVTLGNPAAASMPVSLVMTRKPQLVTVTPDDDVIEAARKMAE
NQVDMLPVVVPHENGSDTPSLEVVGRISKTTMTKVLLDAARKW
>lcl|NZ_CP026520.1_prot_WP_042230610.1_4929 [gene=ppdK] [locus_tag=PC41400_RS24825] [protein=pyruvate, phosphate dikinase] [protein_id=WP_042230610.1] [location=complement(5476384..5479074)] [gbkey=CDS]
MSKQWLMHFYEGNQSLAGLLGGKGANLAEMTRAGLAVPPGFTLTTEACRAYYIESRLPDGLLAEVREAMAELEQSKGQRF
GDAASPLLVSVRSGSVTSMPGMMDTILNLGLNDDTVEGLAAATKNPRFAYDCYRRLIQMFGSVVFGIEGAWFEKKLHALK
QAAGTESDQDIQAQDWQELIGTYKALILKRAGVPFPQDVYEQLRLAVEAVFRSWNNQRAQIYRRLNGIPDEQGTAVNIQA
MVFGNLNEESGTGVVFTRNPSTGERVLFGEYLTNAQGEDVVAGVRTPQPIAKLAGDMPAVYDKLVGMAELLERHYRDMQD
IEFTVENGTLYMLQTRNGKRTAQAAMRIGVELVQAGILTKEEALLRVEPSHLDQLLHPSLDESEAPAALATGLPASPGAA
CGQAVFTADEADAWARAGKAVILVRTETTPEDIHGVLAARGVLTSRGGMTSHAAVVARGMGKPCVCGCELLAIDSEEKQA
SIGGTAIREGDWLTIDGATGRVMIGEVPLKEADMSAELQLMLEWADDVRQMKVYANADNPEDAAKARSFGAEGVGLCRTE
HMFLSPQRLPIVQEMILADSDAARGAALEKLLPMQQEDFDAIFRAMAGLPVTVRLLDPPLHEFLPNLEELVARHAQLTAA
RSGASGPDAELQKLEFLIRKVRGLHEVNPMLGQRGCRLGILFPEIYSMQIEAIFGAALACLNDGVDVSPEIMIPLVGHAN
ELRIMRGLVDEVADSMLGDLQAQVNYRVGTMIEVPRAALTAPQIVQHADFFSFGTNDLTQMTFGYSRDDAEGKFLTHYVD
SKLLPDNPFQVLDRDGVGQLIELAVTRARAAKPKLKTGVCGEHGGDPKSIRFCRETGLDYVSCSPYRVPLARIAAAQAEI
EIAAEEAAAGQQRIVS
>lcl|NZ_CP026520.1_prot_WP_042230612.1_4930 [locus_tag=PC41400_RS24830] [protein=GapA-binding peptide SR1P] [protein_id=WP_042230612.1] [location=complement(5479043..5479234)] [gbkey=CDS]
MSKAKSDVMLGTLLCKWCNEIVGELDTEKVTLYYTQCKDGACRAAVDKEGVGHNEQAMADALL
>lcl|NZ_CP026520.1_prot_WP_042230614.1_4931 [gene=glpX] [locus_tag=PC41400_RS24835] [protein=class II fructose-bisphosphatase] [protein_id=WP_042230614.1] [location=5479506..5480486] [gbkey=CDS]
MQRELSLEIVRVTEAAALAAARWMGRGDKNEADGAATAAMRESFRSVSMKGTVVIGEGEMDEAPMLYIGESVGSGDGPEV
DIAVDPVEGTELVSKALNNALAVIAAADKGKLLHAPDMYMEKLAVGPALCGKLSLDEPLEKTLRSAARLLDKNIADLTVS
ILDRERHAEHIAVLRRVGVRLKLLSDGDVAGAMAPAFPESGVDLYIGSGGAPEGVLAAAALHCLGGELQGRLMPSGEEDA
ARCRAMGIDDPAKLLRLPDMIGDDDVIFAATGITPGDFLGGVRFLSAERAETHSIVMRAKTRTIRYVRSIHHLPSKPVAL
AVAVVG
>lcl|NZ_CP026520.1_prot_WP_241688731.1_4932 [locus_tag=PC41400_RS24840] [protein=S-layer homology domain-containing protein] [protein_id=WP_241688731.1] [location=complement(5480857..5481777)] [gbkey=CDS]
MKKMTNIVLLASLLVFSLGSSAFAFSDLKGSPDEAAIQSLQEKGIVEGMDGDRFAPNTSMTAAQGVQLIVKALDLNLGKI
RFIRAPKASDSFTKAKDDVWYSESFIIAAMNSLPIGRDIDPQQTLTREQFAEWLHAAVEKTGPYPVVNMFVNIADQAEVT
PGSMNAIQVLLLTKIAKLDDASKFNPKKPITRAEAAVMAYNAMEFVAKHSKAEEPATAPSEEEGVKVIVEKQSDELNKVT
LSRGLQPNPGYGITIDRVDYVSDSEAVAYYTLRKPDPDKMYPAVMTEAKASVTVKAGYKVTAKPSK
>lcl|NZ_CP026520.1_prot_WP_042230616.1_4933 [locus_tag=PC41400_RS24845] [protein=M23 family metallopeptidase] [protein_id=WP_042230616.1] [location=complement(5481922..5482926)] [gbkey=CDS]
MKRTATIWGMAAVLLVLPGLAGCGRTEAEPSSAVSPAPPAASPDAEKEAKFEPENLAALFLQGKIKEIYSRFSPEFKKDV
SLEQLSELAGDFNKDVRAYKLQSKLPLGDGDRYVWTDESGQKGLTAAIDGKGVIQGLQFVPLKTYPASDEAFSRAEFSLP
FRGEWTVFWGGTNVLLNYHYEHESQRYAYDLVQIKGTSSYTGDPGKLESYYAYNQEVTAPAPGIVKAVENGRPDNPIGQT
DEKNPAGNHVIIDHGSGEYSFLAHFKKGTITVKPGDKIERGQLLGRVGNSGNTSEPHIHFHVADGTDLFGSKSVRIRFEG
GAAPKQGDVLKGVR
>lcl|NZ_CP026520.1_prot_WP_042230618.1_4934 [locus_tag=PC41400_RS24850] [protein=response regulator transcription factor] [protein_id=WP_042230618.1] [location=5483165..5483923] [gbkey=CDS]
MKEARLLIVDDEPALLHMLEKVLAKEGFTRIDTVSTAEEALEACKRQTYDLIVLDVMLPGKSGFDICPLLRAETDAPILF
LTARGSDLDKLTGFALGGDDYVTKPFNPLEVAARIKARLRRHVRTAPEAGDPRAADAAAEEASEHYDYGRFQVHVQAGEL
RVQGQSLACPALVFQLLVFLCRHPNRIFSKQQLYEQVWGTDSYGDDNTVMVHIHRIRERIEENPGDPRFLVTVRGLGYKL
VRPAAPQEVRVP
>lcl|NZ_CP026520.1_prot_WP_042230619.1_4935 [locus_tag=PC41400_RS24855] [protein=HAMP domain-containing sensor histidine kinase] [protein_id=WP_042230619.1] [location=5483920..5485668] [gbkey=CDS]
MTVRKRLIVRFVIQLVTVSSFILLLIGGAVAYMLTGMAKLEMNRDFSKAGMSMLVQTVRTENGKLIFDEDLLDKALRSGG
WLQAVDETGAVIRSYGTPADVPAQYEPGEFIDYWTGNKPFPYTLFLWIEEKDGRTYTLLYGKRSAENVLLDEILAAGPPG
DKQPEIPASIRERLDRSGAILQLLDRSGRELASYNRPDSIPAVYTLQDLALRRQYSERYGFASASRSLEDGRTWIVHAPV
RSAEQNVFPATEAQVMLTAFVGLAAGLLVLLVGAAFWYGHRFGTPIVHILAWLHRLSRGEYTEPGVAKGRSGSRRRDGRL
KRRYRLYGDVMQSMEHLSGALSSAEEARGRLERTRDEWISGISHDLKTPLASIKGYAHMLETPQYTWTEREIREFAAIIK
DKADFMDELIGDLSLTFRLSNDERPQAAEQTELNEFVRRCILGVVNDAEFAHANVSFEPEEEALVRSVDRQALRRILDNL
LANAVRHNPPSTQIRVFLRKAPEWAASSPGGFILRVEDNGVGMDKQTLGKLFDRYYRGTNTEGTTRGTGLGMAIARQLAL
RLGGSIAAESEPGRGTSVTVTL
>lcl|NZ_CP026520.1_prot_WP_042230620.1_4936 [locus_tag=PC41400_RS24860] [protein=purine/pyrimidine permease] [protein_id=WP_042230620.1] [location=complement(5485930..5487237)] [gbkey=CDS]
MKLFFSTIQWLAFILAGSLVAPLAVGNALHLPPADIATFMQRTFLLIGVSALLQTLFGHKLPMMEGPAGLWWGIFVVYAG
LISSGALALADGLRQLEMGMLLTGALFLLLGALGWIGKIRKLFTPLVTGTYLLILVAQLSGSFMKGLLGIGYLQDAVDSR
VALPAFAVLLLSMLLPRTRSRFLRNYSILLALLTGWVLFMLLGIAPDVPASDAAFALPGIFEWGLPQWDAGILPTSVLIG
LLLLANLIASIQVVEKVSADKNAAAAPVNYNRTSLMMGVSTALGGLFSAMGCVPISGTAGFILTTGLVQRLPFVLASVCM
VLISLFPPVTAIFTSLPMPVGYAVIFVPFASMIGLALREYHSLKGGDRDYFIIGVSLMIGIGSLFVPASAIAQLPGLVRT
VANNGLVLGMLVCIVLEQFFLYRERSARKKDASAV
>lcl|NZ_CP026520.1_prot_WP_042230621.1_4937 [locus_tag=PC41400_RS24865] [protein=ketoacyl-ACP synthase III] [protein_id=WP_042230621.1] [location=5487639..5488622] [gbkey=CDS]
MNSKARISAVGSYVPEKRLTNSDLERLVETNDEWIVQRTGMRERRIAAPGEYTSHMCISAIDNMIAEYGVSLEDVDLILT
ATNTPDLPFPTVSCLVQAHFGIPSTGALDLNATCAGFVYALHTANAFITAGLHRKILVLGADHMSAITDYTDRTTCILLG
DGAGAALVEAAADGAPGAFLGHHLGSDGTGGVHLYRSGLADRLGEKDLEGAGKLVQNGREVYKFAVGTVPKGVSALLDSA
GMAMSEVDWFIPHSANLRIIESVCERLDFPVEKTLYSLEYYGNTSAATIPLSLHMGLKEGKIKPGDLLLLYGFGGGLVHG
GLLVRWG
>lcl|NZ_CP026520.1_prot_WP_042230623.1_4938 [locus_tag=PC41400_RS24870] [protein=hypothetical protein] [protein_id=WP_042230623.1] [location=complement(5488844..5489296)] [gbkey=CDS]
MTSRLTEDAWKLEMLQALACSQRAMMRVTEAAAALPGAAARGADSSCGEAGTAGGNQPADLLLPPEAFPGGRRPGSAASG
LLQTLAAYHFVLQRKIDLLLQPRRRRSKTAAKPWLNTRERVRSGTRAFVCPGAEEPGGNAEEAAPSAPAG
>lcl|NZ_CP026520.1_prot_WP_042230625.1_4939 [locus_tag=PC41400_RS24875] [protein=hypothetical protein] [protein_id=WP_042230625.1] [location=complement(5489293..5489625)] [gbkey=CDS]
MSREVSYVRILQSVAQMQVSIASMLEAKAAEAEKSKAWICSHLTAQQFATHQDQVQQPLEVHDGLIELIEAITRMEQSLG
KHLQIVIGEQENQGGGGMGDFSDLLGGGNK
>lcl|NZ_CP026520.1_prot_WP_042230627.1_4940 [locus_tag=PC41400_RS24880] [protein=hypothetical protein] [protein_id=WP_042230627.1] [location=complement(5489755..5490045)] [gbkey=CDS]
MEREKDALVRHISRSHLELAKILHYKSEVAAHMAGLIGQIPDKNPAFADNETLLSQSVNVTRNVTSYLSSLADLEEALAT
NLGLVVKELESGEGDE
>lcl|NZ_CP026520.1_prot_WP_042230628.1_4941 [locus_tag=PC41400_RS24885] [protein=hypothetical protein] [protein_id=WP_042230628.1] [location=complement(5490185..5491420)] [gbkey=CDS]
MKRLKTNKKPSGTARGQKPRSRMRTRSGNPGKTGAKAVTSRKGRRIAELAAAGRRLGGLLAAGLPVRSASVDGDEAGSCR
TDEAAARRQMNDLWARHGGVLSGLNPAGPEYAAAASAFAQGYCERVGLRDKGWFPFPSDQRVSAVILLHGAGPEAFTVQD
SPLLRLLGELKRLALHEVILASRDEAVTPQLVQTFAAPRLEAEVKLVGAGGVSAGGGGERLAAAEAAEGDILVFLDASVS
SAAEEIAALAGTVAGGSDAALWDVSAKLPHFHKRPPEVMAKQFVNYALGRRDLRADSLTVLPHALSRQAVKKLGAELGHP
VRAHGTALLEGLAVSSVRLSGGDTRGARPFAWMDRSRASDHLEALRLAMDRLGPRLFYPDNWRRRGYAADTDSPYIAEEP
SVSGDFSRESI
>lcl|NZ_CP026520.1_prot_WP_042230630.1_4942 [locus_tag=PC41400_RS24890] [protein=glycosyltransferase] [protein_id=WP_042230630.1] [location=complement(5491401..5492228)] [gbkey=CDS]
MKKRRKTPLAQSSDVNSLHTYRNDPAPKVSVVIPVYNERRSLAAVIREAYHVHPSVEVIAVVNGSTDGSRDIVEKMGPRM
IAFDTALGHDVGRAVGARAARGDIVLFTDGDIVIPAAKLSAFVKAVENGADVAMNTYTGPVKRDSVHRVILAKHALTAML
DRPDLAGTSLTTIPHALSRRAINEIGAEALAVPPKALVLALHKGLNVKATVYAAVGAKNPRRRKSAGRDPVGDLILGDHL
EALHAWIEAAAFSEGAAEPLRTATGKEGADEAAED
>lcl|NZ_CP026520.1_prot_WP_042230631.1_4943 [locus_tag=PC41400_RS24895] [protein=DUF1796 family putative cysteine peptidase] [protein_id=WP_042230631.1] [location=complement(5492467..5493075)] [gbkey=CDS]
MDLSGIAGKYDLIYSLGGNCTPSIQLKRMNLRSFSGVFDWMVSELEDVSGVLENRFQGFMEPGNLKAVGIDYGGGNLLVM
DRKSRIISAHDFPQNQNHIFHLASYPEFRSRLDRRIYRTLHHLMYGSRLLFVRTNAYYEEACELERVLNVLVKQDYRVLV
INHSNADGITDLQWSLPRTCAVGCYSINPACWDTLFKDVSLG
>lcl|NZ_CP026520.1_prot_WP_241688732.1_4944 [locus_tag=PC41400_RS24900] [protein=DUF1796 family putative cysteine peptidase] [protein_id=WP_241688732.1] [location=complement(5493104..5493736)] [gbkey=CDS]
MMRLQDLAGSYDSVFSLGQNCLPAIQMERHGLRSSAGVLDWMMSDSLTDVNRLLGRRFDGFMELAHMSVVEPVTLQPNYQ
VKDSAYGVISVHDFPVNRNSPEQLFTYGEFKEKLNRRINRFLKQLNEEKRILFIRMHGTYEETAELEQVLNGLVAHEYKI
LLVEYEQVPGLVELDWPLPHTCAVQVPYADIWNFNSDRWWDIFFADISLK
>lcl|NZ_CP026520.1_prot_WP_042230633.1_4945 [locus_tag=PC41400_RS24905] [protein=hypothetical protein] [protein_id=WP_042230633.1] [location=5493909..5494304] [gbkey=CDS]
MYKYYQKVEIDPHTYALPLQAGDQEMFLCPLYLYGITPGSRILFKVNIAWSNPVTYDIGELEIKIRRDHPGGPVVESSED
SRFRGGTASMEGEDEGDAADIVVYYLTVRSQGGKALIEGPIVFEGFVLEED
>lcl|NZ_CP026520.1_prot_WP_042230634.1_4946 [locus_tag=PC41400_RS24910] [protein=hypothetical protein] [protein_id=WP_042230634.1] [location=5494414..5494878] [gbkey=CDS]
MATILDYQTNDALIGGGSLPFPVTNPLLGGVELEVATINLTPADTTNLLRFEGTIGWQPDLLLATPLLPILQVRIRVGSP
TGTVIYETDDSALIGTGVLTTITDNILTVSFQHSIQASVVPTGSQPYFLTVAVRGLGSISIRGPIHFGAFVIAP
>lcl|NZ_CP026520.1_prot_WP_042230719.1_4947 [locus_tag=PC41400_RS24915] [protein=glycosyltransferase family 4 protein] [protein_id=WP_042230719.1] [location=complement(5495102..5496334)] [gbkey=CDS]
MKILLATYWPIPHLGGVWPFMLQLKKRLENWGHQVDLMGNGPDEYKYYIYGTDLKVMKDDLMHMLQAKLNRNALPALHQD
PLVRRVEMDRYCMELAAASFGLQQYDVIHAQDVIASLALNRVRPPSTGLVASIHGSLAREIKILLDEEGRLDSERLRPLA
HLYYRTLEYHGAMAAEKAVTASNWLKNVLSREYGVPAGHIEVSPYGFDMEPFTHKLAAGTPLVRPEGKKLIVCPARLETI
KGIHFLIAALANLKQRRSDWICWLVGDGDKRDELIAQVRAAGLEDDVRFLGHREDVPAILALSDIFAHPSVQDNQPFAVM
EAQTAGKPAVVSNAGGLPEMVAHGETGFVSMVGDTDTLAYHLELLLADDELRLKMGEAARLSGCERWSLDRMTEHFVKAY
LAAGQSRGRF
>lcl|NZ_CP026520.1_prot_WP_042230635.1_4948 [locus_tag=PC41400_RS24920] [protein=NAD-dependent epimerase/dehydratase family protein] [protein_id=WP_042230635.1] [location=5496590..5497501] [gbkey=CDS]
MKALVTGGAGFIGSHLAQALADAGHEVHVLDNLSSGKAEWVPGQAVLHVLELNSPELHELVGRIRPEVVFHLAAQADVQR
SIADPAFDAQVNITGTVSLLDACRKSAVRRFVFASTSGVYGDSVHEKLSEDIPAAPISYYGQSKLAAEGYIRIFHKLYGL
PYTILRYGNVYGPRQTPKGEGGVVAVFLQQLRRGQPITIHGDGGQTRDFVYVRDVVEANLAAAASTGCGLYHVSTGRSTT
IRQLAELIRETRSAPVPVLFGEARPGDIRHSCLDNRRAQEELAWSPRYRIADGLRETCAAFLA
>lcl|NZ_CP026520.1_prot_WP_042230636.1_4949 [locus_tag=PC41400_RS24925] [protein=glycosyltransferase family 4 protein] [protein_id=WP_042230636.1] [location=complement(5497724..5498764)] [gbkey=CDS]
MRLVFPILTLNFGGAQRMLAEITNGLADRGYDVTIVMPAHAEVKFDIRARIVRGRHSITLDADDIPPGDVVVSNFYTTVP
ACVEASARDRSVHVRFSLCYEPTFLPDDHITFPTYHTTPHLIVLSEWQQKLIELNHGIRGHIVPIGLGPFMQNLNLRKST
PPAVKISAVVRNSEGGIAWHRDQDYLVSRLAFVKHHYPDTEINLICPPDEFHTAPSLQRLRDTGSFQFCLPNNDEELRYL
YNKTDIYVSSSLYDSGSLPGLEAMRCGAALVTTYSGGNADYARHERNCLLSYRYQNRLAADIVRLIHDKELRARLALQGL
YDSYQWPWARSVDAFAQAIHDIWNSR
>lcl|NZ_CP026520.1_prot_WP_042230638.1_4950 [locus_tag=PC41400_RS24930] [protein=hypothetical protein] [protein_id=WP_042230638.1] [location=complement(5498820..5499002)] [gbkey=CDS]
MSEKVYVIGDLYNSLFSVQVSNPEYLVDYKLWNQIRAGLPDNYRMPDPVIQHILNSHPNG
>lcl|NZ_CP026520.1_prot_WP_042230640.1_4951 [locus_tag=PC41400_RS24935] [protein=glycosyltransferase family 4 protein] [protein_id=WP_042230640.1] [location=complement(5499035..5500261)] [gbkey=CDS]
MKILLATYWRLPHLGGVWPLMVNMKEKLEALGHEVDVFGNSEDESVFHMPFSGRVFYKEAVRPFLEKKISPELYPILHSS
SWLSHFEMNTLSMELAAAYFGLEQYDVIHTQDVLATRSFRRVKSPRTGLVASLHGSVAHEIKLHLLENNVNLEESLIWRF
QSAIEHLGATSSDITIASSQWLKRLLTTEYSVPEHQVEVFQYGYATGDFYAKMASSATPIYKPPGKKVIIFTGRLIPIKG
LSLLIEAVGLLRTVRDDFVCWIVGEGDQEAELNRQCAELGLLETVQFMGRRNDVPYLLAQSDIFVQPSYVDNQPLSLIEA
QLAGKPSIVSDATGVPEMVEHGRTGLIFPSGNSQQLFTHLYTLLENTSMREAMGREAGQWGAVHWSMDLMVQRLLDVYGR
AIAKAAQA
>lcl|NZ_CP026520.1_prot_WP_042230642.1_4952 [locus_tag=PC41400_RS24940] [protein=hypothetical protein] [protein_id=WP_042230642.1] [location=5500462..5500815] [gbkey=CDS]
MSIYYFDAYEFGPYDSRYIQSFDLVFQNIAQRHFDVLIQGYQGQSRFLLGLYHLDPAPLPGSIAHLPTVYNHNQPFTLHI
HTNLTNPYPLAMTMYAKNEGVLVATFTTNDLTVSDPL
>lcl|NZ_CP026520.1_prot_WP_042230644.1_4953 [locus_tag=PC41400_RS24945] [protein=sialidase family protein] [protein_id=WP_042230644.1] [location=complement(5501185..5502405)] [gbkey=CDS]
MANSQVSPAGTLAQTPSVAANYLTTLNIVAAATDFTTGAAQIAVYRSVDQGISWGPLILPLPAGYIGGESNVVAYGFPNV
FIVASHVFNAAGSGSIAVYTSTDNGITFGAPVIAAQGFTTFINNVQVNAVFDNSQASTYQGSVYLVYVHQFNTDFVGGSV
IFFQRSLDNGQTWLRPALVSPDADYVTRGDVAAGLTGIVYVAYISLTPVTRFEVSRSLDGGATFQTPIIVSPVTLVPNVL
PVTGYGFEVPTGACIAVDNSALSTSGILYAVWQDNRQGYSDVFLSRSFDQGLTWTAPVSVTSAPAGTQNFFPSIDVSPKS
GLITILYYSNQINGFDLDAYVAISVDGGASFFNRRESTASSNPNAGSATPVPTIGNYTDISTIPPYGFIGVWMDTRTGAQ
AIFSGN
>lcl|NZ_CP026520.1_prot_WP_042230645.1_4954 [locus_tag=PC41400_RS24950] [protein=sialidase family protein] [protein_id=WP_042230645.1] [location=complement(5502431..5503654)] [gbkey=CDS]
MPNFQVNASGNPKQEPSLAVNLLDANIMIATATDFTLGPPLTGVYRSQDAGVNWTNSFLPLPPGFTGAEAAFAAYGFPNL
FLIVAHVFPGNSSGTIAVYRSTDNGLSFSAPIIVNPGYGTFINDDWTKVFYDNSQSSPYLGNAYVTYNRQYNVDAGGFSM
ATFQRSLDGGVTWEAPLILSDVMRNTERPDVGIGLQGDLYFAWIDTTPPAAYYIRRSLDGGESFGPPILVSGVSVVPLVL
PVPGYEFRVLTCSNIMADTALVSPGRGNIYAVWQDFRQGYADVFLSRSLDQGLTWSAPIPITDSPAGAQNFFPAITVSPK
TGTITVIYYTNRLNGFDLDVYAAQSRDGGSTFTNIRVTTTSFNPNGDDPVPTPLIGDYIDVGTVPPDNFYAVWTDTRTGA
QTTWGGF
>lcl|NZ_CP026520.1_prot_WP_042230646.1_4955 [locus_tag=PC41400_RS24955] [protein=glycosyltransferase family A protein] [protein_id=WP_042230646.1] [location=5503860..5505059] [gbkey=CDS]
MSDVGIVMPVYLQRPSYLQEAIETILNQTYADFRFIIVIDGVSEQMGPQIRQLTAHDKRVEILENKRNVGVSASLNRGFK
RLFKDPHIRYVTWVSSDNRTHPYFIELLRHELAHAPEEVGLVYSCFREIDADGLPLRDEAFQEEFRRYQNKPKEELLDVC
IVGVSFMYKSRFARKIDGYRMEPVEDYEYWLRLSHLCELRYIPVELVEYRVDSDFSISAKLKGSVEQHRRWRHAFRLAKK
EARERQNIPLETTVYLPVTDASESLPDRLDCLLDQYYSNFQVIVVDVTPGRASAGLLHGIRDPLLSTRFLPGMPPVEAVH
ACLKGLATPFAAIAYRERDFEYMDLQVLTAALRAAPEGAYSVCYNGDRSDLAYRFDPRAGMPATELFRSAKLKELLLPK
>lcl|NZ_CP026520.1_prot_WP_042230648.1_4956 [locus_tag=PC41400_RS24960] [protein=glycosyltransferase] [protein_id=WP_042230648.1] [location=complement(5505138..5506220)] [gbkey=CDS]
MKLLFVFYIPSGGMETLNRQRVIALKRIGISCSLLYLHEGTGMQNLDGVPVYVTKDEKEITELVNRERFEAVIISTVAEL
ILPLRSSGYTGKIIYEAQGLGSVEEAEFVMKRVLPLLTLHADAMLCPRTPHLIRMFETYYARMKRFSFHNCIDMESFRFE
RHPVAPVPIIGWVGRIEPNKNWRSLLEVARLFKTRDRPVEIRMYVDNTLGSEEGEFRKLVADYELGSRLTVQANLPHKEM
ARQYSVIGDSGGVLLSTSRVEGFGYAVLEAMCCRCPVVSTDSDGVRSFIADRVTGRFYPHDDHEEAFRQISEIMDNVPLR
EEIRFRASRHAAEVFSPDLYARQFSCMLGELGIGHVKQHG
>lcl|NZ_CP026520.1_prot_WP_042230650.1_4957 [locus_tag=PC41400_RS24965] [protein=glycosyltransferase] [protein_id=WP_042230650.1] [location=complement(5506264..5507007)] [gbkey=CDS]
MPPRVSVIIPFYNCAYVDQAIVSALNQTYPDKEIIVVDDGSTLHRDKITPYLPWVHYLGKANGGTASALNHGIRHASGDY
IAWLSSDDRFHPHKLAVQLNGLASANADISFTNYEVISEHGHVLYAPNFPTDPGNTEYFNTSTKLYTYMLQYNPINGCTV
LMHKNLFRSYGFFDETCKYTQDLEMWLRLAAAGVRIEFVNEKLTQYRLHGEMGSRNHQPAIQREVDMLRARYNPLLAARI
ASLEAGS
>lcl|NZ_CP026520.1_prot_WP_042230651.1_4958 [locus_tag=PC41400_RS24970] [protein=glycosyltransferase] [protein_id=WP_042230651.1] [location=complement(5507039..5507884)] [gbkey=CDS]
MLPKVTVVIPFYNCPYVDQAVSSVLNQTYTNIELIVVDDGSTLHKEKLEPYMDRLIYLGKENGGAASALNMGFALATGHY
VAWLSSDDLFYPDKIRRQVQFMLERNSIFSHTAFDQIDEAGAVTKANVASRYRYPHEFTRAFLSICPVNGCTVMMRRDLF
RAVGYFDETLRYTQDYEFWIRVLLAGFSLDFLDAPLTLYRHHGGMGTMLYQPAIWEEVMSVRSRYRNDLLRHSRSQRRLL
SGRRHTALRSRRRPAARLRKRPAKAVKSRKGKAGKGVKKHR
>lcl|NZ_CP026520.1_prot_WP_241688733.1_4959 [locus_tag=PC41400_RS24975] [protein=NAD-dependent epimerase/dehydratase family protein] [protein_id=WP_241688733.1] [location=complement(5507922..5508929)] [gbkey=CDS]
MPRILITGAGGFTGRAACDYFVSKGYEVAAVVRNEGPAHAKPGNGEAPVVPHREDVSGVPPGPNGPSVLQEPGSAKPGTG
MPAGKLYACDLTRREDVERLVAETGPDWVLHLAGRNSVTESWQDPAAFLQANLMASVYLLEALRTRPGTRILVAGSMLGA
AEPGEPLPHPYGFSKMLLLYAARSWHRWFSQQIMVALPSNLVGPGPSGGICGLLARKIAALEKGGPSEPFRLSSLGEERD
YLDVRDAVSAYETILGHGEPGAVYRIGSERLRPLGEIVSVFRSLSPAALPLEVENKPRPDPVRPVDVSALKALGWSARIS
FETSLSDALEYYRRN
>lcl|NZ_CP026520.1_prot_WP_042230654.1_4960 [locus_tag=PC41400_RS24980] [protein=SDR family oxidoreductase] [protein_id=WP_042230654.1] [location=complement(5508914..5509756)] [gbkey=CDS]
MKLLVTGGQGMAGHVMVRYLRRLPGFQVAYTTRDGSDRDGLLADARDTARIRGIIATFRPDAVINCIGVLNQFAEADPVN
AYRVNGIFPHDLAEAALEAGGRLIHISTDCVFSGRRGDHTEDDVPDGTSIYGRSKALGEVRKAPHLTIRTSIIGPEIRES
GIGLLQWFLSREGEVGGYTRAYWNGVTTLELAKFVEHALQRPELTGLVHLTAPVKVSKYELLNLFRKQFGRTDLEVRKNA
SVRLNRTLKNTRGDLNYAIPGYETMLKEMDEWMCVQCLES
>lcl|NZ_CP026520.1_prot_WP_042230656.1_4961 [locus_tag=PC41400_RS24985] [protein=polysaccharide biosynthesis protein] [protein_id=WP_042230656.1] [location=complement(5509753..5510739)] [gbkey=CDS]
MFRNTRILVTGGTGSWGYELIRQLLPQDPKEIIVFSRNESTQVAMQRSFEDNRLSFCIGDIRDREALIRATEEVDYVFHL
AALKHVPVCEEQPYEALKTNVVGTQYVIEAAIENKVKKVIYISTDKAANPSNFYGMTKAIGEKLIVQANLLRTDTKFVCV
RGGNVLGTNGSVIHLFHKQIHEKNQISITDKAMTRFFLTLEEAISLLFKASEESVGGEIFVMTMPTCKILDLADVLIEAS
GKTGVAIVETGTRPGEKIHEILLTEFESQTTVLYDEVYLVILPTLEIPGLKDHYASYPKAEISSFSSKDNVMEKEAIRAL
LVKGGFLA
>lcl|NZ_CP026520.1_prot_WP_042230657.1_4962 [gene=wecB] [locus_tag=PC41400_RS24990] [protein=UDP-N-acetylglucosamine 2-epimerase (non-hydrolyzing)] [protein_id=WP_042230657.1] [location=complement(5510729..5511823)] [gbkey=CDS]
MKIVTLLGTRPEIIRLSLIMRKLDELAEKHVVVHTGQNFTYTLSDVFFRELGLRQPDHVLQDRQQSLGAQLATMFGELET
ILNTERPDRVLLLGDTNSALGAVLAERMGIPVIHMEAGNRCYDLTVPEEKNRRIIDAVSTINMPYTKQSKEHLVREGVPS
QRIVRTGNPIYEVLRHYGGQIEKSDVLERLNLTPGGYFAATIHRAENVDHPVHLREIMNGLSGVAERFGKRILCSLHPRT
KSRLEAEDSLSIHPLVEFHEPFGFFDFVKLEKNADCVLTDSGTVQEECCIFHVPTVTVRRTTERPETIDCGSNVVSGLDA
KRIVSATGLMTGMPREWECPEGYLDTDVSAKVVKYVLGGNLHVS
>lcl|NZ_CP026520.1_prot_WP_164977154.1_4963 [locus_tag=PC41400_RS24995] [protein=glycosyltransferase] [protein_id=WP_164977154.1] [location=complement(5511903..5513576)] [gbkey=CDS]
MNGRRIMLFSHLCSRQHITGAEKLLLFMARELKERHDCILVVPGEGVLSRQARAAGIYTLVVPFPLVWEMYTPHAGLAGQ
LDGLPAREPGALAKLLRVLGQYRPDLVISNTCVNVLPPVAAKALGIPVLWMIAEKMTDTAFTPVSAEFIHRHADLVAGIS
EATLAPFGVREAESQIVMYPSWHEETMGGGDWSVLRERRRAGFGIDSGEVLAGYIASDIYPLKGLDHFIGAALELCPAYP
HVRYLIAGKPTDAGFYRHCRDLTVRSGYGGRFHWTAFKKEVASLYTAMDFVVVPSLTEEGFGMTALEGMMFGKPVIAFDS
GGLREVLGHTGNARFLVPKGDTRTLAARMRELVEDPLLRMQTGVLNLSAVRSVFGIEAYRSRQEHLLARLENKLAARQNN
PQPQVPPVLDGVPGPLVIVRGRRPGLYLLRDGIRHPVTGAQRQRLGAGVVKVPKALLDSRPVGTVLSFPDEAAPPPRPGA
KLARRKSVRAARKRRPGAKLARRKSVRAARKRRPGAKLARRKSVRTARKRRPGAKLARRKSVRTARKRRPGAKQARR
>lcl|NZ_CP026520.1_prot_WP_042230658.1_4964 [locus_tag=PC41400_RS30125] [protein=hypothetical protein] [protein_id=WP_042230658.1] [location=complement(5513877..5514209)] [gbkey=CDS]
MNKKTGAKSARVKKRRTAGLKKASGTAAAKRRKAGVVLPKGIGGKKKSAKAGRGKGLRKKRLLSASGRRRTAVGAAHSQA
FNEGYDSGFQAGFSQGVADGEHILAEAAAG
>lcl|NZ_CP026520.1_prot_WP_042230659.1_4965 [locus_tag=PC41400_RS25005] [protein=glycosyltransferase] [protein_id=WP_042230659.1] [location=complement(5514347..5515465)] [gbkey=CDS]
MAALSKRSRSFAVKQAEDWQKGYALGAQAGYRLGRCESIMNRISLAPTAFWDKKVLYVTSGKGLPYSPLDTGIIETIRPM
VRELVFAQPVPDLALTVAREMPDFVLVLDGMMLPMDQLELIRKWGIKTAVWLLDDPYYTDVTLNLALSFDYVFTIELNCV
SFYAESGCPNVYYLPLAVDSSVYRPKAVAAGNNRGACFVGSGYWNRVQIFNSITPTLLKYDVHFTGLWWDRLAAYSKLSG
KINLNKWMGPEETADYYNGSKIVINLHRAYDDEQFNNNSRRIQAVSPNPRTFEISACGVLQLTDIRNDLSRFYEIGTEIV
TYGSAAELRDKLEYYLTHEEEGREIALRGMSRTMREHTFTNRLTELFRVVFG
>lcl|NZ_CP026520.1_prot_WP_042230660.1_4966 [locus_tag=PC41400_RS25010] [protein=glycosyltransferase] [protein_id=WP_042230660.1] [location=complement(5515494..5516606)] [gbkey=CDS]
MRKRVKRGRHPVKTPEKARSEGYIRGWEQGCRYGSSRAVLNRLPAPAFGFFEKRVLFVQQGFEAIDAGIQQGLAALTREL
IVCAPEHMLQTAAASRPDMMLVLNGLHVFPEDHLAQADGVRGLGIRTAIWFADDPYFMDLTPAIVPHYDYIFTHERNAVA
YYRSLGCERVYHLPLAAAGHIYYPQPAEPSYLSDVCFIGNAFPNRIDFIDALAPALTGRKVVLIGALWDRLKHYKQFRRG
IHLGWKPMDEAAKYYNGARIVLNLHRAPLDPVYSKNKIGLPAGSINPRSYEIAACGSFQLTDLRGDLAELYTPGAEIETF
TSPQDCAGKLLYYLRNEERRREIAVRGLMRTRASHSFEARLSQLLQIVFG
>lcl|NZ_CP026520.1_prot_WP_042230661.1_4967 [gene=rfbD] [locus_tag=PC41400_RS25015] [protein=dTDP-4-dehydrorhamnose reductase] [protein_id=WP_042230661.1] [location=complement(5516599..5517468)] [gbkey=CDS]
MKILITGAGGQLGSDLVRVLTPRHKLAALTRQELDVGDGAAVDAAVFREQPEIIIHAAAYTDVDGAEREIDRAYRVNSAG
TWHMAEAAERIGAKFVYVSTDYVFDGCKGAPYIERDKPNPLSVYGTSKLHGEKFASMLCERHFIVRTSWLYGRGGNNFVA
KIVEKARRGAELRLVDDVFGSPTYSYDLAVFIGSLLETEEYGTYHGSNEGVCTRYEFAREIIRVLGIEGVKLRPAKADMF
PLPAARPAHSALADTAAGAIGVKPLRPWKEALRHFLLTDLADGREASHA
>lcl|NZ_CP026520.1_prot_WP_042230663.1_4968 [gene=rfbB] [locus_tag=PC41400_RS25020] [protein=dTDP-glucose 4,6-dehydratase] [protein_id=WP_042230663.1] [location=complement(5517465..5518481)] [gbkey=CDS]
MAVLLVTGGMGFIGSHFIRFWHNRHPQDSLINLDLLTYAGNPYNLTDLQDSPRYRFIRGDIADAPFLERLMGGTSIDAVV
HFAAESHVDRSIREPATFVRTNVLGTQQLLEAARREGVPKFVHVSTDEVYGTLGTSGVFTEQTPLAPNSPYSASKAGSDL
LARAYYETYGYPVAVTRCSNNYGPMQFPEKLIPTLIIRALRDLPLPLYGDGEHVRDWLHVGDHCAAIAAVLESGRPGEVY
NVGGGSERTNLEVARTVLKALGKPDTLISFAPDRPGHDRRYAIDASKLTGELGWKPAYTFEEGLRSTVDWYVRNRTWWER
ILSGEYRGHHLLKGGGPA
>lcl|NZ_CP026520.1_prot_WP_042230665.1_4969 [locus_tag=PC41400_RS25025] [protein=sugar phosphate nucleotidyltransferase] [protein_id=WP_042230665.1] [location=complement(5518481..5519203)] [gbkey=CDS]
MKGVILAGGTGSRLRPLTNLMNKHLLPVGKYPMIHYSVMKLKEAGVKEILLVTGKQSAGLYTEYLGSGSQWGVSITYAVQ
EQAGGIAEALGLAERFIGRDKKFVMLLGDNLFQDSLEPFLRDFESQSEGAMVLLKEVPDPERYGVPVLTDGRIVHIDEKP
ARPKSSYSVTGIYFYDAEVFHVLKTCRPSKRGELEITDVNNAYALKGRLAYRILSNWWVDAGTHESLYEAIRQVWKEEEG
>lcl|NZ_CP026520.1_prot_WP_129112534.1_4970 [locus_tag=PC41400_RS25030] [protein=WIAG-tail domain] [protein_id=WP_129112534.1] [location=complement(5519462..5525548)] [gbkey=CDS]
MKKKKNPASGPQRPAPSKYNASMNELRWLDEEPKKKKERADAAVSKAEAEAVRKAERPKPQGLLRAKAEGAAGGGKQTES
NVKGFAKASASPAREKSEIKSVAAESQRDEDLYAGEEMPEDFEESLLVYDTVEERSISEEDDDEVVVASDVPKAIEVPAV
PEAPRAPRAPKAPEVPKTSEIPNVSEKQKAQEVPEDYGNSNRQAQPDTVKTGRNGSPLLLLSERKEERGSEGVVYTDDIV
EAAVTGEKIASRAIDESKMKPNSIGTRSLQDFSVERSKLADGSVTSEKLATGAVLREHLAYGIVNGDHIQIRSITSDKIA
DKSITVEKLADGTIDAAKLAPYSIQSHHLAEASITPELLEDQAVTSDKIKSGAVQSRHLGNYVINSAKLAEDAITTEKLR
HEAVTGEKIAPGSISASHLMPGTISGDKLSSGAVNAVHLAKGSVGDQHIGAGVIGSEHLKPGSVTRETLTPGAVDSAQLA
DEAVGFRELARKSVRPEHLADGSVLSVHIAEGEIRSTHIGSAQITEELIENGAIKSLKIGRRAVESRHIDENAVLTSNLA
NQSVTTSKLATESVSTDKLTDYAVSTAKLNDSSVTSKKLARGAVESEHMQPFSVQSVHIANNAITAAKISEQSVNGSHLK
EGAVSGIHLGSNAVEGRHLAAGSVKQQHLAKGVVTSETLASGAVTGDALSEGAVTKETLAEGAVTSEALASGAVTSKALA
SGAVTSEALANGAVTRETLAAGSVTSEKLASDSVTSKALAYGAVTSEALAPGSVTSEAIAAGSVTREALTAGAITREALA
EGAVTRGVLASGAVTRDALANGSVTSEAIAEDAVTSVALASGAVTSRALAYGAVTSGALGEGAVTSEALAYGAVTSEALA
AGAVTSDELAYGAVTSEALAEGAVTSEALAAGAVTSDELAYGAVTSEALAEGAVTSEALAAGAVTSEALAEGAVTSEALA
YGAVTSEALAYGAVTSKALAEGAVTSEALAAGAVTSEALAYGAVTSKALAYGSVTSEALAARAVTSEELANGAVTSEALA
DGSVTNEALAYGSVTSETLAAGAVTSKALADGSVTSEALADGAVTSEALASGAVTSEALAYGAVTSKALAYGSVTSKGLA
SGSVTSKALASGAVTSEALASGSVTSEALADGAVTSEALADGAVTSEALAARVVTSEALADGAVTSEALAAGAVTSEALA
YGAVTSEALSDGAVTSEALSAGSVTSEALANGSVTNEALAVGAVTSETLAAGAVTSGALANGAVTSEALADGAVTSEALA
AGAVTSGVLASGAVTSEALANGSVTSEALAAGAVTSETLASGSVTSEALANGSVTSEALAAGAVTSEALTSGAVTSEALA
IGSVTSEALAEGSVTSEALANGSVISEALAAGAVTSEALASGAVTSETLAEGAVTSEALAAGAVTSEALAKGAVTSEALA
SGAVTNEALATGSVTSEALAEGAVTSEALANGSVISEALAAGAVTSEALASGAVTSEALAEGAVTSEAFAAGAVTSEALA
KGAVTSEALASGAVTNEALAYGSVTSKALAYGSVTSKALGNASVTSKALAYGSVTSKALTIGAVTSEALASGAVTSEALA
AGAVTSEALANGSVISEALAAGAVTSEALAAGAVTSEALAAGSVTSEALAAGSVTSEALANGSVISEALSAGSVTSEALA
TGSVTSEALATGSVTSEALAAGSVTSEALATGSVTSEALAAGSVTSEALAAGAVTSETLAQGAVTGDHVQPGTITLAHLH
PDALPIRELADQQHGTHLAPNSITEDHLAMGSVASGHLQASSVLSLLIANRAVTGEKLADEAVSSAKLAPSAVGSVHLQD
DAVDTDHLANGSVTEDKLASGSVGAGHLRSQSVENGHLAEGSVTFSKISAGAVQPIHLAEASVTEAKLAPESVHAGHLAA
GAVEESKLAFRPVQAPSSKKAVLQQFGLADFRFEEQDDGLEIDISFEEPFANASYVLVATASHPACYAVCKQKTENTAVL
SIVRTQPGLAFDVVLNWIAVGSKADSAD
>lcl|NZ_CP026520.1_prot_WP_042233311.1_4971 [locus_tag=PC41400_RS25035] [protein=glycosyltransferase family 4 protein] [protein_id=WP_042233311.1] [location=complement(5525670..5526824)] [gbkey=CDS]
MMNISSIAIVSPGSFAVPSGTSSSVEHVVERLADRLSEAAAQVTVLGRRTRRRPAREQIGGVTYVRPSYKSGLPYIEAVY
RELKPMNPDLIQVENRPRFVRYLRKRMNRATICLGLHSITFISKPHIGRKEVAECLRCADLIFVNSDYLRQEVIRRAPSV
ANKVVVNPLGADPSIFAPRMFHEEARQEMLASLGLSGRKVIVFAGRLRPMKGVEQLLHAMPAVFAAHPDAVLLVIGGAGY
GSTKQTAYVRKLKRLAARYPGHIRFIPYVSHDSIPAWFRLADVVAVPSLRIEAFGLVNVEAMSSGVPVVATRVGGIPEIV
LHEATGLLVPLQSVEAELAQAISRLLADDELRRQMGQRSLERVAELYTWEKMAQRQRTFYDLLP
>lcl|NZ_CP026520.1_prot_WP_042233314.1_4972 [locus_tag=PC41400_RS25040] [protein=L,D-transpeptidase] [protein_id=WP_042233314.1] [location=complement(5526838..5527173)] [gbkey=CDS]
MAYSIDINTSTFVLTLSDDGKTVNTYPVGVGKVGTSTPAGTFTIVNKEPNPGGPFGVLWMGLSKKHYGIHGTNDPSSIGK
RSSHGCIRMQNKDVLALSRVVPIGTVVRIHP
>lcl|NZ_CP026520.1_prot_WP_042233316.1_4973 [locus_tag=PC41400_RS25045] [protein=glyoxalase/bleomycin resistance/dioxygenase family protein] [protein_id=WP_042233316.1] [location=5527463..5527843] [gbkey=CDS]
MTKGFTSCLQIFPVPRPGETAVFYERLGFRAVYYLEASEPHVCLYKDAVEIVLTKSKLDEVVPNRVVHGYGYDAYLITDS
QEEIEAELVRLGVKIVRPLSVTDYDNRELVFEDIDGRWIAVGKKQN
>lcl|NZ_CP026520.1_prot_WP_174446843.1_4974 [locus_tag=PC41400_RS25050] [protein=carbohydrate ABC transporter permease] [protein_id=WP_174446843.1] [location=complement(5527944..5528720)] [gbkey=CDS]
MFLVALVWIVPLVWMLWTAFRPKDSVINPTLSWDFTLEHFQHVWSAAPFGLYYGNTLLITGGILVVQLLTVTMAAFAFAR
LNFKGKEFVFMLFLVQIMIPADVLVFPNYNVLKELSLLDTKLGIMLPYFASAFGIFLLRQMFKTIPPELDEAAVMEGCTK
WQILWRIYVPLARPSYVAFALVSVSYHWNNFLWPLIVTNSVENRPLTVGLAIFAQSFETGAQWAEVSAATLIVIAPLLAG
FLLFQRQFMNSFMQSGIK
>lcl|NZ_CP026520.1_prot_WP_042233321.1_4975 [locus_tag=PC41400_RS25055] [protein=sugar ABC transporter permease] [protein_id=WP_042233321.1] [location=complement(5528882..5529772)] [gbkey=CDS]
MTGKVAVWKENGFAWLLLLPSLVFLILFTFYPMLKTLILSFYQADLATPQPIFTGLDNYKNMAEDAVFWKVLRNNVWFAL
GTVPTAIALALVMAAFANKALRAKGFVRTAFFYPNMIPMIAIANVWLFIYTPQYGLLSRGMGWLGQSQAANILGSQDTVM
WAMIAMIVWKEAGYLMIFYLAGLQNIPKDLYESAEMDGVGRWTVFRKITFPLLMPTTLFVMIVALTNSFKLVDHLFIMTK
GGPDNASNLLLFYIYDTAFSFWDQGLASALTMVMIVLLMALAALQYFGLDKNIHYR
>lcl|NZ_CP026520.1_prot_WP_042233323.1_4976 [locus_tag=PC41400_RS25060] [protein=ABC transporter substrate-binding protein] [protein_id=WP_042233323.1] [location=complement(5529863..5531182)] [gbkey=CDS]
MRTIAKMSVLTLLTASLMAGCAKPQTETSASTPGAGTEKAKEKTELTFYYPIAVGGPLTATIEAMAADFTKENPDITVKP
VYTGSYADTTVKTQAGVQGKTPPDVAVNLSTELYSLLDMNAIIPLDEFIAKEDKNWLSDFYPAFLANSQTGGKMYSIPFQ
RSTIVLYYNKDMFKEAGLDPNKPPTTWEEMAEYGKKLKKEGRWGLEIPVSGFAYWMMQTFSLQNGKNLMSEDGKKVFFDT
KENAEGLQFWSDMARKDKIMPEGVTEWATVPSDFIAGKTAMMYHTTGNLTNVKKNAKFDFGVTFLPQKKSYGSPTGGGNF
YIFKDIDKKKQEAAWKFVKFMTEPERAAKWSIETGYVGVRKSAYETDVMKKYVAEFPPALVARDQLQHAAAELSTHNNGK
VSKALSDQLQAVLTGTAEPAAALKKAQEEADKALAPFNK
>lcl|NZ_CP026520.1_prot_WP_042233326.1_4977 [locus_tag=PC41400_RS25065] [protein=HAD-IIA family hydrolase] [protein_id=WP_042233326.1] [location=complement(5531276..5532103)] [gbkey=CDS]
MRGLTEWNEAEAYFFDLDGTIFLGGRLLPGAAELIRSLEEAGKTVGFLTNTTTQTAADCGRRLRGLGLAVRDEQVVTAAA
AAAAYLAEQAPQARVLAVGEEALIAELDAQRIRVVRTPLEATHVVVGMDRRFDYAKLHAASRALRSGAELIACNPDTFCP
VEGDELPDTGAMTAAIEAASGVRASVVAGKPSLHYAQVALRRSGKTADRCVMVGDRLETDIRLGLASGMRTALVLTGAAT
LEEVRAGTVMPDWIVPTLEAFTKLSDEEIFNIPIT
>lcl|NZ_CP026520.1_prot_WP_042233328.1_4978 [locus_tag=PC41400_RS25070] [protein=glycerol-3-phosphate responsive antiterminator] [protein_id=WP_042233328.1] [location=complement(5532415..5533017)] [gbkey=CDS]
MNKGIGNLGPRRIIAAVRHSEDLKLACASDLPVVFLLVGDLFTVQDYVKQVIDSGKKVFLHVDFIAGLGSDPIIIKYLAE
KVRPTGIISTKTHLIKHAKKNGLAAVQRLFLIDSAALENGIANVQQSGPDAVEIMPGLIPRVITEVSERLSVPVIAGGLI
HHHEEIETALQAGAAAVSMGSRALWVKEAKRPRADEAVVR
>lcl|NZ_CP026520.1_prot_WP_042233331.1_4979 [locus_tag=PC41400_RS25075] [protein=amino acid permease] [protein_id=WP_042233331.1] [location=5533533..5534987] [gbkey=CDS]
MKQPNSGSETTELKRGLKARHLTMISLGGSIGTGLFLASGGAIHSAGPGGALIAYAAIGIMVYFLMTSLGELSAYMPVSG
SFSTYASRFVDPALGFALGWNYWFNWAITIAAELSAATLIMKFWLPDSPSILWSLLFLVLMFSLNYLSVRGYGESEYWFA
LIKIVTVIVFIIVGILMIFGIIGGESVGFSNFTLGDAPVSGGFMAVLGVFMAAGFSFQGTELVGVAAGESENPRENVPKA
IRTIFWRILLFYILAIFIIGLLIPYTSDNLISGDVENIAISPFTLVFEKAGLAFAASVMNAVILTSVLSAGNSGMYASTR
MIWTLAREGKAPRFLGRLDKRGVPVVALIATAAVGMFAFLASMFGDGVVYIWLLNASGMCGFIAWLGIAVSHYRFRRAYK
AQGRDLRDLPYLAKWFPFGPIFALTVCIVVILGQNYSAITESGVDWAGLIASYIGIPLFLIVWIAYKVKKKTKVLRLEEC
RFDD
>lcl|NZ_CP026520.1_prot_WP_042233333.1_4980 [locus_tag=PC41400_RS25080] [protein=cation acetate symporter] [protein_id=WP_042233333.1] [location=complement(5535073..5536608)] [gbkey=CDS]
MNLTAFWLFLGIVAITLLITYFAAKKTNSTDEFYTAGGGLTGWQNGLAIAGDYMSAASFLGIAGTIALAGFDGFFYSIGF
LVAYLVVLYLVAEPLRNLGKYTVADMIAARFNEKRIRGVAALNTITISTFYMIAQLVGAGALIKLLLGLDFTTSVLIVGL
LMTVYVVFGGMTATSWVQIVKAVLLMGGTFILSLVVFSKFNFSVTEMFDQLKAATPLKEKFLDPGNKYKIPLDTLSLNLA
LVLGTAGLPHILIRFFTVKDAPTARSSVVYATWIIGIFYVLTLFLGFGAAAFVGYDNIVAEDKGGNMAAPLLAKALGGDF
FFAFIAAVAFATILAVVTGLVLSAASAFAHDFYSHIVRKGKATESEQMKVARWASVGVSILSIILSLAASKLNVAFLVSV
AFAIAASSNLPVILFTVFWRRFNTAGAVTGMLVGLISSLVLVALSPSVWSPEAGKAIFVGTPLFPLTNPGIISIPLGFLG
AILGTVLSKKKSSDAKFDEIVVKANVGNINM
>lcl|NZ_CP026520.1_prot_WP_042233335.1_4981 [locus_tag=PC41400_RS25085] [protein=DUF485 domain-containing protein] [protein_id=WP_042233335.1] [location=complement(5536640..5536951)] [gbkey=CDS]
MKKRVKELNYTEIAQSSQFQALLKHKRQFILPLSLFFFAFYFTLPILTSYTTVLNTPAFGSVSWAWVFAFAQFIMTWVLC
ILYSRRAAKFDVLVQQIRQQMKG
>lcl|NZ_CP026520.1_prot_WP_042233337.1_4982 [locus_tag=PC41400_RS25090] [protein=hypothetical protein] [protein_id=WP_042233337.1] [location=complement(5537068..5537370)] [gbkey=CDS]
MSGLVWGTVVVVPWLLVWLQNKVSGAKFVLDAAAYAALVLFSGIAGITVGEILVNNTVFMTNVHKVFLNPVFLVSGGWLG
LYALQRLIVEGIFSGRIKKR
>lcl|NZ_CP026520.1_prot_WP_042233339.1_4983 [gene=asd] [locus_tag=PC41400_RS25095] [protein=archaetidylserine decarboxylase] [protein_id=WP_042233339.1] [location=5537583..5538374] [gbkey=CDS]
MKPLFRLMTELSSRKWISRMTGSFSKSPLSRSLIPWFAKSYGIPVEDAEKAIGEYATLNEFFTRRLKPGLRPVDTDPAAL
VSPVDAVITGYGPVEAGRMLSVKGQDYTVEELLNRSPRTINYEKGFFMVLYLSPTDYHRIHSPVTGEILEKEHVPGKVYP
VNEFGLRHMTRVLSRNERLITVMEHEAGEIAVVKVGALNVSSIQYVAGLPSHLQAGDDLAYFEFGSTVVLVTENGILTPR
SDLQVGSKVRMGERIGTLKQSTH
>lcl|NZ_CP026520.1_prot_WP_042233341.1_4984 [locus_tag=PC41400_RS25100] [protein=MFS transporter] [protein_id=WP_042233341.1] [location=5538399..5539652] [gbkey=CDS]
MASQATVNAGTARPSAGIPTSKPTVYKVLLAISLVHLFNDSIQSVIPAIYPILKDSMTLSYAQIGWISFYFNLTASIMQP
VIGMYTDRKPKPYMLPLGMASTFLGMLLLAFAPSYAFVLLAVSLVGLGSAAFHPQGSKVAHLAAGSRKGLAQSIFQVGGN
AGQSLAPLLTKWIFIPLGQFGSIWFTLVAGAAMLVQTFIARWYKGVIEAGAVTAKKTVQRAMDPAKRNKIAGAIVVLVFL
VFIRSWYAAAMTSYYAFYLRDVVGTTIDQAQIYIFLFLAAGALGTFFGGPLADRFGQRTVIFVSMCGAAPLAALIPYVNL
GWATVLLVLTGFIVFSSFSVTVVYAQTLIPGKVGMVSGLITGFAFGMGGIGALVLGNLIDAHGISLIMKLSSLLPILGIL
TLFLPSDRKLREWSSEV
>lcl|NZ_CP026520.1_prot_WP_042233343.1_4985 [locus_tag=PC41400_RS25105] [protein=hypothetical protein] [protein_id=WP_042233343.1] [location=complement(5539783..5541240)] [gbkey=CDS]
MKKREEQEWFGLLDEITADLAENHIPDLDTAEFPLEKDPDRLAAIKQRTFEKIAGISPGAKIKTAICPSTGGELAPTAPR
VGRTGTNRPKLLRRYALLAALAALLLTGGVLAGAQPEVAAQMRRVLQYLPGFAGVADSEKTEQRVEYVQPLPTTHKIGKG
QLEIRGLSIGEKNSTVTLVGTGLPRASEVTLRNGLGEKYVFRRATLTGAEGGWTGFYSYEGRIEVAGNTTLIAEGQELPL
KLVAPREVEQVADLGPTAEHSGIRLTAVSAAADEGKMKLTILPQLPGEVKINSYGFNSFYKEVPSAKLFNPGTGEEIPYT
QDETFPNSSEIYWERGKEAVGGAKLVIPALDLTRSYDQPAVIRLLVPEKGSVEVNRTFDLFGFPVSVGRVERILKGAKID
GEGEAGEDSIRIWFELHGDEQAAETLYEFPPSLLNGGAVWHRNEQTDKLEWMLLSVQPGQKEYELKVSSLHTRVKGPWEF
SVDQP
>lcl|NZ_CP026520.1_prot_WP_042233345.1_4986 [locus_tag=PC41400_RS25110] [protein=sigma-70 family RNA polymerase sigma factor] [protein_id=WP_042233345.1] [location=complement(5541237..5541806)] [gbkey=CDS]
MEKSEEWLVRGMSEQNPAALESLMDLYASNIYGLVHRILQGAVSREDIEECVSDVFAAAWNRTAEYEPRRGSYKTWLLIL
AKYKALDYRRRLRPETRSGKMLVETELMAETDTEQIVLDRETQEEFLALIDRLEDKDRIIFYRRYYYYESVEEIAAAVGL
TAKAVENRLYRARKKLKEQMETVRTEGIS
>lcl|NZ_CP026520.1_prot_WP_042233347.1_4987 [locus_tag=PC41400_RS25115] [protein=alpha/beta hydrolase] [protein_id=WP_042233347.1] [location=5542044..5542859] [gbkey=CDS]
MGIYIPVEQGVNLYVEDLNPGGGGKTILFIHGWPLNHTMFEYQFDQLPKAGYRCIGVDLRGFGKSDRPWGGYSYDRMADD
IRRVIETLRLEGIILAGFSMGGAIAIRYMARHGGFGVSKLALFAPAAPLFTKRPDFPYGNSIEELNHLIEGTYTDRPKML
EDLGGRFFARYVSAPLQNWFHGIALSSSGHATAMGAALLRDEDLRPDLPKISVPTVIFQGAQDMICPYPLAQATQAGIKH
SELVPFAYSGHGLFYCEMGKFNRELMRFVGS
>lcl|NZ_CP026520.1_prot_WP_042233350.1_4988 [locus_tag=PC41400_RS25120] [protein=hypothetical protein] [protein_id=WP_042233350.1] [location=5543016..5543687] [gbkey=CDS]
MKRKLGCYHAHYSNIEHIEKALGSYDIELVHFVDPGLDRIKNDSDFTEELALNKIKSTLDWLASCHVDAILITCTYFTAH
LPEHFSCPVPIIKIDDPLFHKILQLNRPSVLVFTNPSTVEGTMEQLHKFAAAAGQEIQVEPVLLENTFQLVMQGKKNDYI
EAVSRGLFHAIEDHPRKTVFAAQLSMVPAAERVGRESDAFLGNHLDALPGYLKDILSLEAVGR
>lcl|NZ_CP026520.1_prot_WP_042233352.1_4989 [locus_tag=PC41400_RS25125] [protein=hypothetical protein] [protein_id=WP_042233352.1] [location=complement(5543714..5544979)] [gbkey=CDS]
MLTITESNLRADREPLLHPFGFKGGQLTELWQPIVTLTSESGHKGTGVGVQSPLWSDANLMASLGEQGANELMYKFAEET
VKSLEGASFAHPMELFEQLYARPSQSWFARQGHPNVRPTYVLNALAAVDFAAWVLYAREREIYRFSDLLPAEYHSLGEQK
TEAVAAVPAIAYGTDDEEITGLLEQGCTVLKIKLGSGGSDMIDWDSKRLRHIHELTQAYLQTRPAPAEVSYYLDLNGRYA
AKDDLLRLLEDADRSGSLPSVILVEEPFLDGREISVEDVPARIAGDESIQSESDARYYMDLGYGALALKPAAKSMSLSLK
VAALAAERGVPCFCADLTANPLLVEWTRVLAAHLRPLPGMAYGLIESNGGQNYRGWDRLVSYHPAAGAGWTRTRGGLFEL
DEAFYAESGGVFRPSSYYENR
>lcl|NZ_CP026520.1_prot_WP_042233658.1_4990 [locus_tag=PC41400_RS25130] [protein=hypothetical protein] [protein_id=WP_042233658.1] [location=5545176..5545433] [gbkey=CDS]
MRTRPIYLFYGGVSLLMMVIYTVQFISGMPEVYLLQTGPLGNVERFENLFSKGLPCAVLALVMTFFYLILYLADRPKKEP
DHEPG
>lcl|NZ_CP026520.1_prot_WP_042233354.1_4991 [locus_tag=PC41400_RS25135] [protein=hypothetical protein] [protein_id=WP_042233354.1] [location=complement(5545533..5545934)] [gbkey=CDS]
MKTIEIRNPLLPLLAGSAFSAFMFATLMIQNWDLSDLLKPLTVLGLLLALFTAGYVLFDLIESRHVEFEGRVTDYNNGRL
SLKTREGKVKRYRLPTREIQPGESGELLEAGNQVKATVTKRTGQLLTLEVLSG
>lcl|NZ_CP026520.1_prot_WP_042233356.1_4992 [locus_tag=PC41400_RS25140] [protein=GAF domain-containing protein] [protein_id=WP_042233356.1] [location=complement(5546200..5546679)] [gbkey=CDS]
MFHTETYGGTREEQYTLVIAQLKALIEGEPSRIANLANASALLGQFLRDINWVGFYLLEGEQLVLGPFQGLPACVRIPLG
RGVCGTAAQKRETLVVEDVERFPGHIACDSASRSEIVVPLVKDGELLGVLDIDSPNLARFDETDRVMLERFVTELAARL
>lcl|NZ_CP026520.1_prot_WP_042233358.1_4993 [locus_tag=PC41400_RS25145] [protein=NAD-dependent malic enzyme] [protein_id=WP_042233358.1] [location=complement(5546722..5548182)] [gbkey=CDS]
MAGFTGGTSIILRLEMMKERTNFGQIATTIGDAGGDVVAIDVNRSTKTVTVRDITVKVSDQNQTDAIVKALNELDGVKVI
NVSDQTFLLHLGGKIETTAKIPIKNRDDLSRVYTPGVANVCMAIHEDPSKAFSLTIKRNTVAVVSDGTAVLGLGDIGPEA
AMPVMEGKAMLFKQFAGVDAFPICLNTQDPEEIISIVKAISPAFGGINLEDISSPRCFEIENRLIEELDIPVFHDDQHGT
AVVLLAGMLNAVKVAGKKLEDCKVIVCGIGAAGIACTKILLAAGVKNVIGVDRQGALVRGESYPYESWQWYAENTNPNEE
KGRLSDVIEGADIFVGLSGPGVLKVEDVKKMAVDPIVFAMANPTPEISPEEAEPYVRVMATGRSDYPNQINNVLCFPGLF
RGLLDCRATRVNEQMKLAAAKAIASVVSEEELNELYIIPSVFNKQVVKKVREAVIAAAYETGVARRKSADIRSDEAAQSA
DGAGML
>lcl|NZ_CP026520.1_prot_WP_042233361.1_4994 [locus_tag=PC41400_RS25150] [protein=PLP-dependent aminotransferase family protein] [protein_id=WP_042233361.1] [location=complement(5548291..5550177)] [gbkey=CDS]
MKILIPFNHYAQLFPHKNTALYRALRESIVEGRLESGSKLPSSRELAEIYGVSRGLVTQVYEMLMSEGYVEGKPGKGTFV
AYRSDQAAGSGSSSDEPLLMSAWGRLLMESGGGAGRREEGSGVSAGNKEDSREQELLGAERRPGGAKAADAGAAGREQTG
GVKQVETGTAGYAEARGTGVAKASGVKVTDAGTVGQEQTGGVKQTEAGTVGYAEAMGEGLAEGSAVKTSDAGSAGQTAAL
ASTRARADETRSRSAGSWSADLHTPAGPGSVVFELSGTDSAAFPRQAWNRCLHEQVRSAAFLESPLRDPQGDYSLRSAIA
LHLRRSRSIDADPDCVAIVNGSMQAIALTMQLLVNPGDEVVVESPGYSGAVRAAAAAGGVPVDAPVDGQGIIPQPWNARL
LFVTPGRQFPTGAVLPLKRRQELLRWAKERGAVIIEDDYDTEFRYAGRPLEPLKALDREGRVLFVGTFSRTMYAGLRLGY
AVLPPSLVEPFRRAKQHYEPLTTGMMEQRALAAFMNSGAYERHLRRMQRVYGEKAALLRSGLTALDPAFEWMPSEAGLHI
FGWWKGGAEAYRRFAAECAERGVFWRAAGPHVRCGERPAACFGFGHLSREEIGRGSAVMTEVLTETFL
>lcl|NZ_CP026520.1_prot_WP_042233364.1_4995 [locus_tag=PC41400_RS25155] [protein=PrsW family glutamic-type intramembrane protease] [protein_id=WP_042233364.1] [location=5550555..5552816] [gbkey=CDS]
MILPGLRLTIRTARDRFRTQWKKRASQAPRLFAFYSILSWLGLALLCLSLPFMKDSRTLLVQALASFYVLWQLWIVCRSK
TVGWSAYAAFFIIGTWIVAPLNTLFIQLVFLPFDPAVYAVWNASLVSPLTEELMKLAPLLLFLRGSRKATSLSLADFALI
GGATGAGLQFLEEYWMRLHASLEPAGSVAVSPDSAMSWNLFHFLPSLEQGLASPWAGHTAATALCALGLGIAIRLRRVIG
KAAYALPVFLLVWVAADHAVHNGTQATWLNGFHMTLGGGAYTKALFIVLLIGAVAYDYADLNRMRKGLTLFRSENLIEPF
RELYRSLSMLQEGAKRYVSLLRFYRIRRRLAFTLLYGKKDAKRSLGNLQDEMKTLNKAAVPVLLAAGFLLLSAAAWTGGL
AAAAGSSPAVSACVSCLYAPTSAWWSGQSAAAKGVLLLGTAFLFTSFFPLWGAVSDTLAGDFKARRKHAGRLQPGEILPL
LGAFTPRMAAAAGLQRLLDRLAGLGAGKKPSAPSAESASSGAQEQPSAAQSAEAPSAAESAADAGSGADDTADAPGAAGA
ARSGTVPERRHRFTTARTRQTVHQGKQAYLPAGTLYGGGWTIMGLHNWNGFLDAVRRDRCTVVRTEELPENGIRRAVIRR
QGTDPLTGETFTSLTYGKTFYPKHYNDSEIDRLGEDALNKAILHNRLQSQNDHRGRLQAYTFREEVVGPDGKIIGVQGRV
LPDAEGNLTEIIDHFPYNHRQTVSFRRLSADQD
>lcl|NZ_CP026520.1_prot_WP_042233366.1_4996 [locus_tag=PC41400_RS25160] [protein=hypothetical protein] [protein_id=WP_042233366.1] [location=5552904..5553230] [gbkey=CDS]
MEANLDLLRHLAQIHRPVDSDANDIWHYMILNMEEDALDPPALQAVASALESETNQGIGFREIQFERVLEHPELVEMEYY
ETNLVCPRESLAAHIRKILLLYDAGMEA
>lcl|NZ_CP026520.1_prot_WP_042233370.1_4997 [locus_tag=PC41400_RS25165] [protein=aminotransferase class I/II-fold pyridoxal phosphate-dependent enzyme] [protein_id=WP_042233370.1] [location=5553513..5554820] [gbkey=CDS]
MNPLAKQLNETIEKENAYVYEMLSSLGKQIYFPKVGILSQSAEAKTKAGKYNATIGTAIEKGQPMHLKVIQDTLSAYDPK
DLYEYAPPAGKPGLRTAWRKKMLDENPSLSNKTFGNPIVTNALTHGLSIAADLFVEAGDRVIIPDKNWENYELTFGIRRG
AEIVTYPLYNEDNTFNSAGLREQLLAQKDKGKAFVVLNFPNNPTGYTPGAEEGRQIVAVIREAAEAGINVVAVTDDAYFA
LFFENSLQESLFGQLADLHDRVLAVKVDGATKEEYVWGFRVGFITYAGKNEALLSALEQKTLGIIRATISSCSHPSQTFV
LHALNSPEFNAQKQEKFEIMKGRANKVKALLDSGKYGDSWDYYPFNSGYFMCLKLKTVDAEALRQRLLEVYGVGTIALSE
TDLRVAFSCIDEENIEELFDLIYQGVRDLETVSAK
>lcl|NZ_CP026520.1_prot_WP_129112536.1_4998 [locus_tag=PC41400_RS25170] [protein=hypothetical protein] [protein_id=WP_129112536.1] [location=complement(5554992..5555174)] [gbkey=CDS]
MQEKSSCKIRRSGLVRELDGPSAVLEIEGRTVRVPAGKIAAGTAAGTPVQWNGTLWIPFK
>lcl|NZ_CP026520.1_prot_WP_241688734.1_4999 [locus_tag=PC41400_RS25175] [protein=metallophosphoesterase] [protein_id=WP_241688734.1] [location=complement(5555321..5556130)] [gbkey=CDS]
MVEGTFIWNRMRMMTVLVWLLPAAACAYLFLIVPTMWLKVERVRVPAGLGIKILQISDVHVEKLRVRPGRLARLIDEEKP
DYIALTGDFTRRSKYLPKVEKMLKTVASRGIPAFAVLGNHDHRLDPAGMRKLTAILDEAGVVLLNNRSVDLGAFQFVGVD
DFGTGHSRIGKAFRHTAAHKPVVVLTHDPNLVLRLKRPFTYLMAGHFHGMQFHIPFLFRFIRKGELAARGITKGLHETSY
GPYYISKGIGQAGPNARFMIRSEVTVHEL
>lcl|NZ_CP026520.1_prot_WP_042233375.1_5000 [locus_tag=PC41400_RS25180] [protein=DUF5317 domain-containing protein] [protein_id=WP_042233375.1] [location=5556451..5556969] [gbkey=CDS]
MIYFIVLAVVITLLSGRNPLSMIEKVNFRLPYLLLGSFAAQIVIFILAARTNRTYPYWTEAAILAVLLFLWLNRKVSGIS
LIFLGAIWNWTALVIHGGLMPVSDTALAWAGLGSLPENEPRHQLMEASAFWWMGDWIPLVRRVISIGDVGIGAGLVRFML
GNSVKRRKNEGP
>lcl|NZ_CP026520.1_prot_WP_042233377.1_5001 [locus_tag=PC41400_RS25185] [protein=EAL domain-containing protein] [protein_id=WP_042233377.1] [location=5556956..5558968] [gbkey=CDS]
MKDHNPFTSLLSAFGILTLITASAFIAQNDYDLKILLPIAVMVIVFEMMPLKMPNGTYFSGVTIGFILSLFEMGFFASTF
FLVLNTLTFFTLHSKPFYKIRWFRFFTTLGMYGVSMLAAWGMGVLTAEAPLFVKIVCVFLAFEGVNLLLRTGIMRSVTGA
PFSADKIIYEARSIQIGILIASVVLYKMLKQPSYELFISEIVFTIIAVKIIHYFVSGYLKQIDIIEESNKRYKKMAYYDS
VTGLPNRAHFNERLTQDIAVAQETKGKVGLLFLDLDQFKNINDTMGHVSGDALLVQVAGRLQSCLGQDHTVSRLGGDEFT
IIVRDIAGAQDCVLVAETVLSALKTGFFLQGKDIYITPSIGISLYPDNGYDSETIVKNADTAMYRAKELGGNLYSMFTSD
LEETIEKKLILQRNLHKALERGEFHLVYQPQLQLDTGQIQGIEALVRWNNPELGLVSPEDFIPLAEESRQILSIGEWVLR
TACGQNKAWQNEGFPPVRMAVNLSSIQFEHDELIPAIRRILDETGLEARWLELEITESLLLRNKIKTMRTLRQLKELGVH
IAMDDFGVGYSSLSYLNYYPFDRLKIDKSFIHHMNDRTGDEWIVRTVIQLAHGLHMTVVAEGVETEEQLTYLKKQSCDCI
QGYLISKPLAPSLLQEKLWGGSGTPETRPD
>lcl|NZ_CP026520.1_prot_WP_042233379.1_5002 [locus_tag=PC41400_RS25190] [protein=LLM class flavin-dependent oxidoreductase] [protein_id=WP_042233379.1] [location=5559242..5560255] [gbkey=CDS]
MLKLGILDQSQIAEGRTASDALKETTRLAQEADRLGFTRYWVSEHHASKSLAHSSPEVLLAHLAAVTSRIRVGSGGIMLP
HYSAYKVAENFRLLEALHPGRIDLGLGRAPGGMPLATRALQADKYVSIDRYPDQVADIAGYLHETMPPEHPFAALQASPS
IPTAPEVWLLGSSGESARIAAKQGAAFGFAQFFGSPGGEEAMRHYKKHFQPSVLNDKPRTLAAVLAVCADTEEEANKLAS
STDLYFLRLERGLELGFFPSVDTALNYPYTEYDLARIKERRERRIVGNPEQVKRRLHAVAEAYSADELLIVSPIHNFGAR
LHSYRLIAGAFGMKGEE
>lcl|NZ_CP026520.1_prot_WP_042233382.1_5003 [locus_tag=PC41400_RS25195] [protein=S-layer homology domain-containing protein] [protein_id=WP_042233382.1] [location=complement(5560547..5561752)] [gbkey=CDS]
MKKLNMKKTVLSTLVLAMAMGGGTAAFANGNGKDKDYDKDKDWKEGKNGSVHGSFDFRITFDDLKGSDVEWALKNIASLS
SKRVFEGYEDGSFKPRNTVTRIEAITAAVRLMGLRDKAESSEEKQTKLNFKDANKIPDWAVGYVAVALENDLFGENEDMV
QPSKPADRLWATILLVKAMKLDAEAKAKINTQLSFKDAKQIPAGSVGYVAVAIEKGLIDGFEDNSFRPNMPVTRAQLAAL
LDRTDSQMPGQDQTTAVGKVTGTVSGNTLVLDLNGTLKTYALHPEVFVYRNGAKVSASQLQNGDVVRLRSYNNTVVFIEV
TKKAVEDRTLQVSGLFDSMTLNKDGRIATVTVTNQVYGSADKATYKVASDVVIQGDVSRLVKPTPVELKGKDKVVETIVI
K
>lcl|NZ_CP026520.1_prot_WP_042233384.1_5004 [locus_tag=PC41400_RS25200] [protein=hypothetical protein] [protein_id=WP_042233384.1] [location=complement(5561923..5562471)] [gbkey=CDS]
MKIIRRIIVVLLVLAVAAGLATWGLLTYIKPQQQLDLAYTELSLESQLVDLIRSKKPEVRLSDEELNQLGKKAIAKHARV
SPDVTITGADFKRNGENLTADVNLVWKNTVPVGAQMTFKLAWTGKELTVTHTGTHIRSFEVPPGLVSLAPVSISLDSYLP
SIVHIKEVRFDPDFVVIGLTLR
>lcl|NZ_CP026520.1_prot_WP_042233387.1_5005 [locus_tag=PC41400_RS25205] [protein=DEAD/DEAH box helicase] [protein_id=WP_042233387.1] [location=complement(5562590..5564113)] [gbkey=CDS]
MYDNFEQLQIAPVWAEKLKERGIGRPTAVQAEAIPLILTGRDAIVQSQTGTGKTLAFLLPALQRIDVNAKRVQAAIIVPT
RELGMQILRVAEDLIGDGPIKCQALIGGAALARQVEKLKKGSPHLIIGTPGRINELIKARKLNVHHVSMLVVDEVDHVLE
LGSMKEVYSIFKALQRSSQSLFFSATIPAEIGQAAESLMNDPAEIRINPDQRTASTLEHLAFNCEARNKIDTLRRIIRLY
EPPKAMVFVNATADVSEVVSKLQHLGLSVEALYGEAGKQQRANVMQSFREGKFQLLLATDVAARGLDIEEVTHVINLDMP
LNAEYYLHRAGRTGRMGRSGTVISIVAPKERFIIDKFRKSLRISIDEKEMYEGRVVDAGQGPRSGFRGAPRRDDAGDAAR
VSGTARSASGGEAGARRGGASGAGGERLSTDRRGDSGRSGAPARTGRGSASAASRSGAGEGRSGAARGDSAQQPVSRKAA
RKQETAQDRKNKGAPKWLKAKRDGDGK
>lcl|NZ_CP026520.1_prot_WP_009676277.1_5006 [locus_tag=PC41400_RS25210] [protein=sigma-70 family RNA polymerase sigma factor] [protein_id=WP_009676277.1] [location=complement(5564266..5564847)] [gbkey=CDS]
MSKYMIHLFTSNYFHLDKSLQEKIYKEFHVMVYPTIYFILQDHNLVEDIIQESFLRAIDKAPQLKETEKYEGWLKKLTRN
VTLNYLQKHKRNRNELDSEITLTLRETAPAYEAGSTPLEKEVELKIMQEAINHYVNQLSPAYRQILAMKVVHNMSYKEMA
EEMNVTEGVIRQRLYRAREAIRQMLLEEWDFLR
>lcl|NZ_CP026520.1_prot_WP_042233389.1_5007 [locus_tag=PC41400_RS25215] [protein=prolyl oligopeptidase family serine peptidase] [protein_id=WP_042233389.1] [location=complement(5565080..5565988)] [gbkey=CDS]
MSESIQSGKKISTLSPVTLSPAVRAAGNEAESLKDGRLLERELLSRHARNVSLYVVTYAAQGLRVKGYLAVPDHAEPLPA
VVYCRGGIRRVGMVRIRRIVSMARRGYVVFAPFYRGNEGGEGREDFAGEDRYDVYHGLLMLRDLPEVASVPVSMIGFSRG
ALMALLAARDCEEVGPVVVWGGVSDLHDTYEERVDLRRMLKRVVGHPLKQADEYIRRSPARWADLIESPVLIVHGTADPQ
VNISHAHKLAAALEHAGKDHTMLIYEGLEHRFPKEMDEVALDEVFRWIEQKRIEIEGDKSKK
>lcl|NZ_CP026520.1_prot_WP_042233391.1_5008 [locus_tag=PC41400_RS25220] [protein=DUF2573 family protein] [protein_id=WP_042233391.1] [location=5566128..5566385] [gbkey=CDS]
MSQDFENEFNALVSKAAELLTGDGSSEMSEKIKIWAVYQHLHKAMPALTNHWNGSHPDAKAQVRAIFEEIRDLNERQKAS
RGKPE
>lcl|NZ_CP026520.1_prot_WP_042233393.1_5009 [locus_tag=PC41400_RS25225] [protein=YkuS family protein] [protein_id=WP_042233393.1] [location=5566443..5566691] [gbkey=CDS]
MSKIAVEDSLSSVKQALQSSGYEVTSLNQADGASCVVISGLDQNVMGFADATTPASVINAHGLTDEEIVQEVGRTTGQNQ
SF
>lcl|NZ_CP026520.1_prot_WP_042233394.1_5010 [locus_tag=PC41400_RS25230] [protein=SDR family oxidoreductase] [protein_id=WP_042233394.1] [location=complement(5566799..5567572)] [gbkey=CDS]
MNIAGKVVVVTGASSGIGAQTARALAQRGAVPVLMARSRDKLERLAAEIGGCGVYRLDVQDTAEVNRVFAEVFAEYGRID
VLVNNAGFGVFERFLDAPLEQFEEMMDVNYMGIVRCTKAVLPAMVKSGSGHIVNIASMAGKIGSAKSSGYSATKHAVLGL
TNSLRQELAGTGVRLSAVNPGPIDTPFFERADPTGNYVKNISWFMLKPERVAEAIIRAVEKEKAEIDLPFIAGIGTKLYS
LFPRLLDKTAQRMLNKK
>lcl|NZ_CP026520.1_prot_WP_042233396.1_5011 [locus_tag=PC41400_RS25235] [protein=chemotaxis protein CheX] [protein_id=WP_042233396.1] [location=complement(5567756..5568211)] [gbkey=CDS]
MKAEYINPFLESARIVIEQVANVRPTTGKLSVKDIYFIERYVWIQIGMTGQMQGNIRFGFEESTALRMVSAMMGGFILTE
LDEMGHSAISELGNMISGNASTILYNQGIKVDITPPKVVTGDIHEVDALKALTVPLLMDGIGELDIQVLLA
>lcl|NZ_CP026520.1_prot_WP_241688735.1_5012 [locus_tag=PC41400_RS25240] [protein=TIGR01457 family HAD-type hydrolase] [protein_id=WP_241688735.1] [location=5568618..5569397] [gbkey=CDS]
MHGFLIDLDGTLYKGNEAIPGAKEFVDLLNKNSLPYLLVTNNSSRSPEHVAGHLNGMGIEVGPDRVYTTAMASAQYLLEH
GSGNRVYCIGEEGLQSALLEAGFELTDKEPDYVVQGIDRQLSYDKLASAVKHLLGGARYILTNPDHLLPSDSGLIPGAGS
IGALIRTASGANPVVIGKPSPVIMNYALARLELQAKDTWVIGDNTATDIGGGIAAGCRTALVLTGLATEANVREHISKSG
NEPDLICRDLPHFWNSLMK
>lcl|NZ_CP026520.1_prot_WP_042233400.1_5013 [locus_tag=PC41400_RS25245] [protein=LysR family transcriptional regulator] [protein_id=WP_042233400.1] [location=5569645..5570547] [gbkey=CDS]
MNLNQLETLITISKTMSFRKAGELLNLTQPAVSAQIRSLEEEFGTILIDRNQPVTLTDSGKLFLEHAERILRTVEDLKQR
LADLSQTPQGHIHIGTTTSIAMQILPRVLFYFQDQFPLIKTTIHSMTSSQIMASVDNGSVDIGISYLYEKYPSLEASVLY
YDTFELIVSTSHPLSRYSHISIEKLRNLPFIMLSPETAGRRFVDQVFKTYDINPQIVMELSSSEEVKRMVELNLGAAIIS
KLSVVNDLDHGTLKMVKVDELDITHPVGVVYKSGRYLNSAMHQFLQDLKGMPETQFLGSE
>lcl|NZ_CP026520.1_prot_WP_042233402.1_5014 [locus_tag=PC41400_RS25250] [protein=histidinol-phosphatase] [protein_id=WP_042233402.1] [location=5570559..5571365] [gbkey=CDS]
MKFDLHTHNERCGHATGTIRHYVDAAVDAGLSVIGISDHSPYFYSEQDQLFPGIAMAKSAFAAYVEEVLGLKKEYADKIE
VLLGVESDFFPEHLDLYRSMYEKYPFDYIIGSVHLSGGVSIFNKNRWKGLDDAEHIVHKETYYDLIQQSARAGLFQILGH
IDAMKGFYPAFSDIPTPKVDETLKVIADEDVAIEINTSGKTKTVGGWYPSDLILERALHYGVKVTFGSDAHVPQRVGDDF
ELVSKRLKEIGFKEWCFFRNKERVLVTL
>lcl|NZ_CP026520.1_prot_WP_042233404.1_5015 [locus_tag=PC41400_RS25255] [protein=cation diffusion facilitator family transporter] [protein_id=WP_042233404.1] [location=5571980..5572960] [gbkey=CDS]
MTDFHHLDHVKEQSTSKRTLWITLLLTLFFTIVEVVGGIMSNSLALLSDSAHMISDVIALGLSMTAIYLASRKPNSRYTF
GFLRFEIIASFLNGLALAVIAIGILIEGIKRFYEPQNIDFKLMLIISSIGFVVNLVLTLVLHRSMEEEENLNVQSALWHF
IGDLLSSVGVIISAIIIYFTGMLWFDPLISIVISCIIFIGGSRIIKESYLILMESVPAKFNLEEIRRDIGNVEGVEDVHE
MHLWAISTDHYSLTAHVFIDDKIQPFCIILAINETLKEKYGITHSTIQMEHAGIHPHGEYGREFLKHREEKHQEDSLIPV
QKQAMT
>lcl|NZ_CP026520.1_prot_WP_042233406.1_5016 [locus_tag=PC41400_RS25260] [protein=zinc-dependent alcohol dehydrogenase family protein] [protein_id=WP_042233406.1] [location=5573104..5574102] [gbkey=CDS]
MKAVLLETFGAPESFSIGEIPTPAVRPGHVLIRVAASSVNPIDARIRQGLMPNLAPPAPAVLHGDVAGTVEAVGEGVTKF
RPGDEVYGLAGGFKGFGGALAEYMLADAELIARKPASLSMSEAAALPVVALTAWQGLFDRARLQPGQQVLVHAAAGGVGH
IAIQLAKWGGADVFATASSGAKLDIASRLGADGVIDYRTEPVEDYVRRCSDGGQGFGVVFDTVGKDNLDRSFAAARTGGH
VVAIAARSTHDLSPLHAKGLTLHVVFTVLPLLTGVGRKHQGETLERLARLADEGHLRPLLDETRYTFRDAAGAHRRLESG
SAIGKVVLTNDL
>lcl|NZ_CP026520.1_prot_WP_042233409.1_5017 [locus_tag=PC41400_RS25265] [protein=EamA family transporter] [protein_id=WP_042233409.1] [location=complement(5574325..5575209)] [gbkey=CDS]
MWKAIILVFLGACSYGMLSTFVKFAYEAGFEPGEVVGAQNLFGTLISLVLVLTLSRHKMKPGQWVPLLMVGLTTGSTGIL
YYNSLQTVPASIAIVLLFQFTWIGVILEAVLERKRPSRAKAVSLVLLLIGTGLASGVADLGSIDLPVIGVVLGLLSAVTY
TGFIYFSGKVSTEVPAINRTLMMNMGGLLLVFVYHTPSFIWNGALVEGLWKYALLVGVFGIVIPTVCFNYGMPKLGTGLG
SILGAAELPVAVVLSASVLQESVSLVQWAGVLLILFGIAYPHLRRGKAGELKPI
>lcl|NZ_CP026520.1_prot_WP_042233411.1_5018 [gene=sdhB] [locus_tag=PC41400_RS25270] [protein=succinate dehydrogenase iron-sulfur subunit] [protein_id=WP_042233411.1] [location=complement(5575363..5576133)] [gbkey=CDS]
MAEAQTAVPSKTVKFIVTRQDSPESAPYKEEFEIPYRPNMNVISALMEVQRNPKTADGGETTPVCWESNCLEEVCGACSM
IINGKARQACAALIDKLEQPIRVEPMRTFPVVRDLVIDRTRMFNALKKVKAWIPIDGTYDLGPGPRMAESKRQWAYELSK
CMTCGVCLEVCPNVNEKTSFIGPAAVSQVRLFNSHPTGEMNKDERLDTLMEDGGIEGCGNSQNCVRSCPKGIPLTTSIAE
MNKDTTKRLFKKWLGM
>lcl|NZ_CP026520.1_prot_WP_042233413.1_5019 [gene=sdhA] [locus_tag=PC41400_RS25275] [protein=succinate dehydrogenase flavoprotein subunit] [protein_id=WP_042233413.1] [location=complement(5576136..5577908)] [gbkey=CDS]
MANSKVIVVGGGLAGLMATIKAAEAGVPVQLFSLVPVKRSHSVCAQGGINGAVNTKGEGDSTWEHFDDTVYGGDFLANQP
PVKAMCDAAPGIIHLMDRMGVMFNRTPEGLLDFRRFGGTKYHRTAFAGATTGQQLLYALDEQVRRFETAGLVTKFEHWEF
LGAVIDDDGVCRGIAAQDLRNMEVQTFSSDAVIMATGGPGIIFGKTTNSVINTGTAASAVYQQGVYYANGEFIQIHPTAI
PGDDKLRLMSESARGEGGRVWTYKDGKPWYFLEEKYPAYGNLVPRDIATREIFHVCNDLKLGINGENMVYLDLSHKDPKE
LDVKLGGIIEIYEKFTGDDPRKLPMKIFPAVHYSMGGIWVDYNQMTNIPGLFAAGECDYSVHGANRLGANSLVSAIFGGM
VAGPKAIEYIQGLDKHVEDMSTAMFDREKQKHTDRYESLLKMDGKENAYVIHKELGELMTSSMTVVRYNKNLEETIVKIK
EYKERFQNINMTDTARWNNQGVAFTRQLWNMLELSEAMTLGALLRNESRGAHYKPEFPERDDDNFLKTTKAKWTPEGPVI
EYEDVDVSLIAPRKRDYTTDKKKKEKEGGH
>lcl|NZ_CP026520.1_prot_WP_042233415.1_5020 [locus_tag=PC41400_RS25280] [protein=succinate dehydrogenase cytochrome b558 subunit] [protein_id=WP_042233415.1] [location=complement(5577901..5578578)] [gbkey=CDS]
MGKNSYFNRKLHSLLGVIPVGFFLIEHLLTNFEATKGPEAFVDQINWLNSLPLVLVLEIVGIWIPLLYHAVYGLYVAFTA
RNNVSRYGYFRNQMFLWQRITGVLTFLFVAWHFFETRFQVAIGNVEHERLGQTMHDIVSQPLLLTIYVIGVIAASFHFTN
GMWSFLVSWGITVGPRAQRVSSYIWIGLFLVMSVMFIASLVAFKDPQFQELPVVSGMIGGVTSNG
>lcl|NZ_CP026520.1_prot_WP_042233417.1_5021 [locus_tag=PC41400_RS25285] [protein=LysR family transcriptional regulator] [protein_id=WP_042233417.1] [location=5578871..5579770] [gbkey=CDS]
MEHLDVFAVVVEQASLNKASQVLNLSQPALSRKIMKLEEQLGVQLFSRKGKRLDLTRAGEIAYEYALEQRQGERRFLQTL
YEYKTAGRHMSLTIGASLTTLQTTLPELVTMYLAEYPSTDMKALTGKTHEIVMMVKEKKVDIGLVASSIDQPGLACEPLF
DDHLCLVLPQGHPYLEREELSIADLNGLSMILFSKGTWYRVLTDELFHRFFIFPDVKMEIDSFEAIMRLVVTCQAATLLP
RSYLHDHSSDHDGLCVRELPELMNTPRTTSLIFVKETHQNEAVAHFIRKAKAHFAGLSQ
>lcl|NZ_CP026520.1_prot_WP_156972861.1_5022 [locus_tag=PC41400_RS29985] [protein=hypothetical protein] [protein_id=WP_156972861.1] [location=complement(5579825..5579971)] [gbkey=CDS]
MKKRASPGLAGPGGSVQPGSGGTTGGSVEYQTGYDGLAVRAVIAVMRA
>lcl|NZ_CP026520.1_prot_WP_042233662.1_5023 [locus_tag=PC41400_RS25290] [protein=sodium:proton antiporter] [protein_id=WP_042233662.1] [location=complement(5580039..5581316)] [gbkey=CDS]
MENALTQLIDYELYLFILVVVLGMLFGKLAGWLRLPDVALFLIAGMIFGQAFHWIAEPSSSLTNQFILVVGSTLILFDGG
RNIRLSGLKHVWISVTLLSIPGVILTCGAVAAASHWFLGLPWLHALLLGAIISSTDPATLIPVFKQVRIRTKVRETVESE
SAFNDATGSILTFSLIAILLGTEHVSAASGAWQFVKTALGGLAVGAIIGYAGAFLTAHVKIGFLRDYATILMLAAALGSY
LAGEHLGVSGFMATFTAGLIWGNSDVFKLHMEDKKQEMTHFSENVTVLMRMLIFILLGSQVNFPLLLKYLWPSLGIIFVF
MFIARPLVVLLCTLPDRKAKWTWREIVFMFWVRETGVIPAALSGMVAGLGIAHADLIASVTFMAVLITILFQASTTAFVA
RKLGLEEDSGGGGAHGVPSAKPGSH
>lcl|NZ_CP026520.1_prot_WP_009676247.1_5024 [locus_tag=PC41400_RS25295] [protein=TrkA family potassium uptake protein] [protein_id=WP_009676247.1] [location=complement(5581341..5582003)] [gbkey=CDS]
MAFIQYAVIGLGRFGSSLSQELTQLGHEVLGIDRDEEKVDEMSDKLTHTVVADATDEEVLRSLGVRNFDCAVVAIGDNIQ
ASIMAAILLKELGVKTVVAKALTDQHAKVLEKIGVDRIIFPERDMGVRVAHQLVSPNLLDYIELSKNYTIAELVVSRRIS
GLTLKELDPRAKYGCSIVAINKTDDVIIAPTAEDVLHEKDVMVVIGTKEQIDTFESRAIR
>lcl|NZ_CP026520.1_prot_WP_042233420.1_5025 [locus_tag=PC41400_RS25300] [protein=TrkH family potassium uptake protein] [protein_id=WP_042233420.1] [location=complement(5582018..5583361)] [gbkey=CDS]
MLKKSRFTPPQILVLGFALIILIGSGLLMLPVSSTDGLPLPFIDALFTATSATCVTGLVVVDTGTHFTTFGQMVIISLIQ
IGGLGFMSMATLFAFILRKKISLRERLLLQEALNQGSMEGIVRLFRKVLIYSLTIEGIAAVLFTARWSVDLGFPKGLYYG
IWHAVSFFNNAGFDLFGPITGPFTSLTAYAGDFITNAVTMALIVLGGIGFIVMSDVMDYRKTKRLTLHSKVVLSATSILI
VTGALVIFVFEYSNSKTLGSLDFGGKILASFFQSVTPRTAGANTLDLASLRQASQFFMIILMFIGASPGSTGGGIKTTTF
TMLVGAIVAMVRGKEDIVFFRNRLGKERILKAITLTMFALFLVIFVTMLLSTTEDHEFLMILYEVTSAFGTVGLTLGLTT
KLTVFGKVMIALTMFAGRLGPLTLAYALQPKAEKELYRYPEGKITIG
>lcl|NZ_CP026520.1_prot_WP_042233422.1_5026 [locus_tag=PC41400_RS25305] [protein=CPBP family intramembrane metalloprotease] [protein_id=WP_042233422.1] [location=5583750..5585408] [gbkey=CDS]
MKKYGPDKPLLLLALTGFLLYAAVVFIYNAGAAPAGSGPYETAASITKEQAAEAAVAWLRQRQGADAENGSGAVTVGGSP
GTFVSYQSEKELSGYLTKDKLQTLYDKSYASRVPVDYYRVEVAEASPSVRKWIIDIHMTTKAVIGWSRIGGEALAAVSAS
DAEKAAREAIARHGYEEAEFKRISPPAAMSGMLLFEREKEPVGDAKLTIGVKLDGTSVSSFTQSFRLPAAFTSWLKHQED
TASMWTRISLWGNGLMALIALGYTVLKWRTVDFSRGVLLTAVYAVIYVMNNINMYPAYKAGSTPTSSGQEAILSIMLINF
VTLLMAAAVYFSFTAGEALWRREGKSVMPRWRDADFGARITRSMVQGYLLCLFLLGLQQVLFLVAEKGFGVWSVNDPMSS
PRNMLQPALLPLMAWAAGISEEATFRLLGIALFSRLLRFKWLAVLLPSLIWAAGHTAYPIYPVYTRLIEVTVLGLVFGYA
FLRYGFLTALFAHTAMDSILMGLSLLTGEGLAASTPALGLLYMALPAAVALAVSWLHKLRFVSGSRQNRRPV
>lcl|NZ_CP026520.1_prot_WP_174446844.1_5027 [gene=uvrC] [locus_tag=PC41400_RS25310] [protein=excinuclease ABC subunit UvrC] [protein_id=WP_174446844.1] [location=complement(5585656..5587677)] [gbkey=CDS]
MTREESEDHIRNKLALLPDKPGCYLMKNAEGTIIYVGKAKVLKNRVRSYFTGSHDGKTQRLVSEIRDFEYIVTSSNIEAL
ILECNLIKQHYPRYNVLLKDDKSFPYIKITHEEHPRLEVTRKIVKDKGKYFGPYPNAFAAQETKKLLDRLYPLRKCKTIP
DRVCLYYHLGQCLAPCEHDITQQQNDEMVQGIVRFLNGGHEQVKQELTSKMHAAAEDLNFERAKELRDLIVNIDAVMEKQ
KITTTDVVDRDIFGYVADKGWMCVQILYMRQGKLIERHVTMFPYYGDEYEDFITYVMQVYSENPALPKEIFLPGRSQQEL
PAGSLSAGMEAAAGAAVPQAGEDGGSPESLREAPEEAALPSPLAAEAEEHAGSDSRSMPTTAEQKDVKEALESWLKVKVH
FPQRGLKKQMVDMAKDNARVGLEEKFRLIERDAQRTVLAVENLGSWLGTGPVTRIEAFDNSNIQGSDPVAAMVVFTDGKP
DRKEYRKYKIKTVKGPDDYESMREVIRRRYERVLKENLALPDLIIVDGGKGQMSVAIDVLENELGLFVPVCGLSKDAKHK
TAMLLAGDPAEVVPIPRDSQEFYLLQRIQDEVHRFAITFHRERRAKSMVVSQLDAIPGIGEKRRKLLLKHFGSVKKIKEA
AVEDFRPLGIGEKLAAQILTALRGEQEQEEPSI
>lcl|NZ_CP026520.1_prot_WP_009676258.1_5028 [gene=trxA] [locus_tag=PC41400_RS25315] [protein=thioredoxin] [protein_id=WP_009676258.1] [location=complement(5587841..5588155)] [gbkey=CDS]
MAIVNVTDQSFKGEVEGEGTVLVDFWAPWCGPCKMIAPVLEELAKEEESLKIAKVNVDDNPESASRFGVMSIPTLIVFKN
GQPVDKVVGFQSKDALKNVLSRHQ
>lcl|NZ_CP026520.1_prot_WP_009676287.1_5029 [locus_tag=PC41400_RS25320] [protein=YqzM family protein] [protein_id=WP_009676287.1] [location=5588415..5588561] [gbkey=CDS]
MSDPKHPELHLNEEPRNDLMDVALGFGGFFAVLMIIFIIGVIIKLVIG
>lcl|NZ_CP026520.1_prot_WP_042233664.1_5030 [locus_tag=PC41400_RS25325] [protein=globin-coupled sensor protein] [protein_id=WP_042233664.1] [location=5588756..5589763] [gbkey=CDS]
MIQLSDDRMKQVRFTGITENDLQLLKANEPAFKEVVEEVVEQLYEQIAAEPELRDLISRHSTLERLKQTQADYFLSMASG
VLNEQYIQNRLIVGRVHSRIGLTTEWYLGTYMLYLDLSLRHFKRVLPDGAWLEVVHSLTKMFNLDSQLVLEAYEHDEKAK
IQQLADDQAIMLSGISKAVQDLASMMVELSSSSQIVAETAVQTAESQDKSHQYLKDLEEEVGHIHAMGALMKEVSEQTHL
LGLNAAIEAARAGDAGRGFEVVANEVRKLASRSRESLELIEDKLTAIRTILGTVHGESEQTSIYARTQAASSQELSSFVQ
MVEQVAHDLERLKKK
>lcl|NZ_CP026520.1_prot_WP_042233428.1_5031 [gene=dnaI] [locus_tag=PC41400_RS25330] [protein=primosomal protein DnaI] [protein_id=WP_042233428.1] [location=complement(5589893..5590843)] [gbkey=CDS]
MESLSDMLKQMPKSSWMQRAEEQAKQVLEDPLILKLRGQYPQLDDHTLKINMNRLYQYAREHKNCLNCPGLERCPNDMQG
HYTMLNIEDNGGRTYIHEEKTACRKFIAKQSQDAIRSRIRTFHIDEQALSREHTYADILRADMERAKAVDFLNDYVMRTT
EEGLQKKGLYLQGSFGTGKTFLMCYMLYQLAKKGFTGAIVYMPDFAEDLKGMFGEPQKLKETIELLKETDLLVFDDIGAE
NLNPWLRDHVLGAILNYRMNRKPTFFTSNYDLDALEKHFSFTNKDGDEEYKGQRIMDRVRPYVEVVTVDGYNKRGA
>lcl|NZ_CP026520.1_prot_WP_042233430.1_5032 [locus_tag=PC41400_RS25335] [protein=DnaD domain protein] [protein_id=WP_042233430.1] [location=complement(5590877..5592385)] [gbkey=CDS]
MRLTNLLHFTENHRFCVYRDFALSSLDYKMLSSIYQPMIGGYAVSLYHTLYQQVPSDKTGYAGLEQQRKLFLSLELDPGE
RSRKLFIEQSSKLEAVGLLQTSRIYMQDEDDYIYEYTLFCPLSPNEFFRNQHLTLLLRDKIGKFLLLSLRDELTMPEPAE
LTGADSENLSVPFYELFRLNTQVIDYELEQALYEASAGKQAAQALDVTTKGYAYADLITRYPRESFNRIYVENLKYKPET
MISINMVARKYSLTLQETVRLLDEDGMFDEDGELQIDSLQYKANLYFRQGKKREEERERFLHRAESGDSGGNTADSEEGA
RSVEMQYYLEVPAFLQGSHNEHQYNSLLKNEPYTLVLQSFFTQGPVPNGVLDIFEKIDLNYKLKEEVINVLIHYIHIDKR
SWAKSSIEAVASDMLGKQIASFEQAVEYVRQRLQFRQQAAAKAAAAKASGGYAGKGKARTAGKQMKPQIPIVQNKGQTSR
VTEEELEAMLRKAKKLDEMFNS
>lcl|NZ_CP026520.1_prot_WP_042233666.1_5033 [locus_tag=PC41400_RS25340] [protein=PspC domain-containing protein] [protein_id=WP_042233666.1] [location=5592652..5592876] [gbkey=CDS]
MNKLFRSRTDRKLTGLCGGLAQYLNVDATIVRLIAVIGAVFSFGTFTLIYIIASLFVPKEPYGGYGYHDSYHSY
>lcl|NZ_CP026520.1_prot_WP_042233432.1_5034 [locus_tag=PC41400_RS25345] [protein=YuiB family protein] [protein_id=WP_042233432.1] [location=complement(5592971..5593234)] [gbkey=CDS]
MIQLVIVIVLMFVLFFGLGFILNMLMKTTWFPVYAFVALIIGLSFYYSDKMTSITVVDIAPVIGGLIGAVVSGTTIRTLR
IKGFKMF
>lcl|NZ_CP026520.1_prot_WP_042233434.1_5035 [locus_tag=PC41400_RS25350] [protein=heme-dependent peroxidase] [protein_id=WP_042233434.1] [location=5593428..5594174] [gbkey=CDS]
MSEAIETLEGWYALHDFRRMDWNAWKEASPEERKRAEDELRSFIGEWATIEDNKEGSTALYSIVGQKADILFMQLRETLE
ELNDLETAFNKSAFAKYTIPVYSYVSVVELSSYMAKPGSDPMENPEVVARLKPTLPKAKHICFYPMNKRREGGDNWYMLD
LDSRKAMMRSHGMIGRSYAGKVKQIITGSVGFDLWEWGVTLFADDALQFKKLVYEMRFDEVSARFGEFGDFYVGNLLDND
RLGRLLDV
>lcl|NZ_CP026520.1_prot_WP_042233436.1_5036 [locus_tag=PC41400_RS25355] [protein=hypothetical protein] [protein_id=WP_042233436.1] [location=complement(5594424..5594702)] [gbkey=CDS]
MRHCSVQVRGLLTRSELDRYNALIEVGHYLENQKEYGHVAVVQKEIDILIFPAIERLKEKSRQRDRDTEDYLRRKAEMEA
EMRRIAEEDDED
>lcl|NZ_CP026520.1_prot_WP_042233438.1_5037 [locus_tag=PC41400_RS25360] [protein=NAD(P)/FAD-dependent oxidoreductase] [protein_id=WP_042233438.1] [location=complement(5594825..5596018)] [gbkey=CDS]
MSRVPRIVILGAGYGGVVTSLRLQKLLNYNEADVTLVNKHDYHYITTHLHMPAAGTDKPDNARVNISSLIDEFKVDFVKS
TVVQIRPGEKKVILEDGTLSYDYLVIGLGGEPETFGIPGLKEYAMTIRSINSVRHIREHIEYQFARFKQEPDRKELLTFV
VGGAGFSGIEFVGELADRIPELCKEFDVDPALVHIYNIEAAPTALPGFDPELVEYAMRLLTDKGVTFKIATAIKECTPEG
VLIAGDEFIKAGTVIWTGGIRGNHLLEDAGFETMRGRIKVDEFLRAPGYDNIFVTGDCSLVMNPEGRPYPPTAQIATQQG
ENCAFNLVASIRGNALKEFAFVNRGTVASLGKGEAIGIVGTKKMKGGTAALMKKVIDMRYLYTIGGLPLVVRKGKLR
>lcl|NZ_CP026520.1_prot_WP_275938058.1_5038 [locus_tag=PC41400_RS25365] [protein=NAD(P)/FAD-dependent oxidoreductase] [protein_id=WP_275938058.1] [location=5596486..5597421] [gbkey=CDS]
MFAAFYGGMRQASVTLIESMPQLGGQLAALYPEKHIYDVAGFPKVTAQELVNSLAVQMNHFPIHVCLEEKVLQVVKKEER
LFEVITDKESHLGKAVIITGGVGAFEPRRLELPEASRYEKTNLHYFVSDLNSFAGQKVLISGGGDSAVDWSLMLEPIAEQ
VTLIHRRDKFRAHEHSVEKLMNSRVNVVTPTEITNLEGDDAIRKVTLKHVKTGEETVLDVDAVIVNFGFVSSLGPISEWG
LQIENGSIVVDSRMETNVPGIFAAGDITTYPGKLKLIAVGFGEAPTAVNNAKVYLDPSAKLSPGHSSNMKF
>lcl|NZ_CP026520.1_prot_WP_042233442.1_5039 [locus_tag=PC41400_RS25370] [protein=hypothetical protein] [protein_id=WP_042233442.1] [location=5597523..5597777] [gbkey=CDS]
MSTFCTVCNGMDMLSAVCPHCGSGSASDCGRVNDYFGPYSPYRPIEEISMTNGYPDVTRSECLHMLYCEACGASSVLAVP
ESCT
>lcl|NZ_CP026520.1_prot_WP_009676266.1_5040 [gene=sda] [locus_tag=PC41400_RS25375] [protein=sporulation histidine kinase inhibitor Sda] [protein_id=WP_009676266.1] [location=complement(5597818..5597973)] [gbkey=CDS]
MRQISDETLVESYFKALDLELESEFVDLLLGEINRRHIVLEAYHSDEAALA
>lcl|NZ_CP026520.1_prot_WP_009676280.1_5041 [locus_tag=PC41400_RS25380] [protein=hypothetical protein] [protein_id=WP_009676280.1] [location=5598415..5598576] [gbkey=CDS]
MIVYDIVVNGEVKETIRPQKLRLKELYAYMLEQSKQMKMKYGENTRVLKRILY
>lcl|NZ_CP026520.1_prot_WP_129112578.1_5042 [gene=mscL] [locus_tag=PC41400_RS25385] [protein=large-conductance mechanosensitive channel protein MscL] [protein_id=WP_129112578.1] [location=5598670..5599095] [gbkey=CDS]
MIGEFKKFIMRGNVVDLAVGVIIGAAFNKIVTSLVNDILMPPIGLLLGKMDFKNLKLPLSGGVTINYGAFINNIIDFLIV
SAVIFLMIRQMNKLRRKEEPQEEVKSEPTKECPYCISEIAAKATRCPRCTSVLEPEALPQS
>lcl|NZ_CP026520.1_prot_WP_170318715.1_5043 [gene=erpA] [locus_tag=PC41400_RS25390] [protein=iron-sulfur cluster insertion protein ErpA] [protein_id=WP_170318715.1] [location=complement(5599277..5599636)] [gbkey=CDS]
MIKISEEATDKIKEMLAAEETPDLFLRVGVKEGGCSGFSYGMGFDNDQKEEDKVLDFSGLKVVVDPESAKYLYGVEIDYK
DAGMGGGFTIDNPNAVASCGCGASFKVKGEEGKAEKCDD
>lcl|NZ_CP026520.1_prot_WP_016423556.1_5044 [locus_tag=PC41400_RS25395] [protein=aspartyl-phosphate phosphatase Spo0E family protein] [protein_id=WP_016423556.1] [location=complement(5599739..5599942)] [gbkey=CDS]
MQQSEEWECRQKLDVRIEQLRSQMVENGMKYGFLHPSVQHDSRRLDKLILRYYQLERGESPLETKLE
>lcl|NZ_CP026520.1_prot_WP_009673165.1_5045 [gene=mqnE] [locus_tag=PC41400_RS25400] [protein=aminofutalosine synthase MqnE] [protein_id=WP_009673165.1] [location=5600317..5601420] [gbkey=CDS]
MNVIIDSPDRRMAEIAEKVENGERLSREDGLFLYQSDDLLTIGQLANKVNLRKNGKNVYFVQNMSLYFTNVCEEHCAFCH
FRRDEGEAGSYTLTPSEMLTYISENFHPDIREFHISQGHNPHVPFQYYVDALRTLKQHYPQVSIKAHTAAEIDFFARISG
LSYKEVLRTLMDVGLDTLPGGGAEILTERYRKKMRVEKADTNQWLDVHRSAHELGMKTHATMLYGSIETLEERIEHMALL
RDLQDETDGFLVFIPNSIQPASASAGIRRRVPAVDELKTIAISRLMLDNFPHIKAYFINMGIKVTQLAFQFGASDAHGTI
VKERISHAAGAQSPDGVTKEDLVWLIKGAGRVPVERDTFYNVVQTYN
>lcl|NZ_CP026520.1_prot_WP_009673186.1_5046 [locus_tag=PC41400_RS25405] [protein=NAD(P)/FAD-dependent oxidoreductase] [protein_id=WP_009673186.1] [location=5601500..5602564] [gbkey=CDS]
MNKFVILGGGYGGITIATELLEKEIPDNWTVTMVDRSPFQGLKTEYYALAAGTAAETELRVAYPNDPRLSLKYGEVTEVN
LETKQVIFADKEPLDYDYLVIGLGCVDNFHGIPGAQEFSNGIQTFSQTRITYQRVNDVVPYGQVTIVGGGLSGVEMAAEL
RESRQDINIRILDRGASILSSFPTKAKEFVRDWMLDHDIEMRSHVSLKRLEGGDLYNEEEIIRTDATIWTAGIKPSPIVE
QLAVDKDPSGRVLLNEYHQIPSHPNVFVVGDCASLPFSPSAQAAESQGKQIVEVMQALWKNETPKLGQIKLKGVLGSLGK
KTGFGLMGKKTVMTGMVPRVLKNGVLWMSKHHWG
>lcl|NZ_CP026520.1_prot_WP_042233448.1_5047 [locus_tag=PC41400_RS25410] [protein=YuzB family protein] [protein_id=WP_042233448.1] [location=complement(5602620..5602874)] [gbkey=CDS]
MRPIIEFCASNMHHGTDAVMKKLEDNPDYDVIEYGCLGNCGECYLFPFAMVNGEIVAAETADKLFDEISKKIKEVEEIYD
MFGD
>lcl|NZ_CP026520.1_prot_WP_241688736.1_5048 [locus_tag=PC41400_RS25415] [protein=hypothetical protein] [protein_id=WP_241688736.1] [location=complement(5602970..5603215)] [gbkey=CDS]
MMNENNDCFHCYGDGETECQSCSGEGSLGDRGICTSCGGSGKTACARCSGSGVVADAFDVQDSITDIMRSLYVRPEYTRG
R
>lcl|NZ_CP026520.1_prot_WP_009673094.1_5049 [locus_tag=PC41400_RS25420] [protein=NifU family protein] [protein_id=WP_009673094.1] [location=5603420..5603668] [gbkey=CDS]
MTENTQSATMYDEVLEVLDKLRPFLQRDGGDVELVDVEDGIVKLKLMGACGSCPSSTITLKAGIERALVEEVEGVVEVVQ
VF
>lcl|NZ_CP026520.1_prot_WP_042233450.1_5050 [locus_tag=PC41400_RS25425] [protein=SDR family oxidoreductase] [protein_id=WP_042233450.1] [location=complement(5603724..5604512)] [gbkey=CDS]
MAEVRRTALITGSAKGLGKMAALTLANQGYDIVINYVHSEEEAHDLVRHIQNTGVRAAAIQGDVSRGEDIERIAETVGET
MGGVDVLVNNAGPFIRERRLFADYTRDEIDYLIRGNLLGVIQLDHALLPHMRRKRWGRIIHFGFGHAAEARAWPHRAVYA
AAKVGLVSFTKTLAEEEAAHGITVNMICPGDIRGANKEKRIEEVENCTDDETPRGRPGTGEDVARVLSFLCDPKSDFLTG
NIMEVNGGLDPIQTLPMLKRGK
>lcl|NZ_CP026520.1_prot_WP_053228776.1_5051 [locus_tag=PC41400_RS25430] [protein=Cthe_2314 family HEPN domain-containing protein] [protein_id=WP_053228776.1] [location=complement(5604421..5605242)] [gbkey=CDS]
MLRTIFNETKRSERSDERLEQAFTSIKRYLNVLQQSRHSSPSYSPKKLHRLYVWSQGFLDALDELEESKYCAGRFAERVK
KSYLDEMQPEELDDYRRYVYFYKNALIRLFSVLDKLGYFLNDLFDLKTETLKTHFSFFTVLRKMRESGIHPALQQKLYEL
KTEAKQPLQVLRNQRNMEIHYVNVEMLDDLMLSQSYLGDRIQVENIADNLAQMNRGYEMVCITMDLVFRYISRGSIGENG
EENHGGGTQNGSHHGQRERARQNGGVNAGQPRV
>lcl|NZ_CP026520.1_prot_WP_042233451.1_5052 [locus_tag=PC41400_RS25435] [protein=COX15/CtaA family protein] [protein_id=WP_042233451.1] [location=complement(5605697..5606728)] [gbkey=CDS]
MYKLTKRLSFLTMIGMFLIVIMGALVTKTDSGLGCGNEWPLCNGKFVPAYTISSVIEYSHRAVTGLVSLLLLATFILVFK
TGSRKDAKYFVAGAGFFTVLQAIMGALAVVFSQSSAVMALHFGLSLLAFAMTLLLWLTFTKWGDYTVSVARTTDGRKSAL
PKNAAPIGRGFRAFVWFVGVYCYIVVYVGAFVRHTESSGGCIGWPLCNGQVVPELTGATGIVFFHRIAAVIMFIFGLALF
LIARKKYSHLENVYKGAGWVLGFLIAQIFSGATVTWSLGHDTAFLFTGMIHAVLICALFGWISYLCVVMLNARVPLGAEH
PRNKQESPQPAKAPVNRKEAVTE
>lcl|NZ_CP026520.1_prot_WP_174446842.1_5053 [locus_tag=PC41400_RS25440] [protein=thioredoxin family protein] [protein_id=WP_174446842.1] [location=complement(5607026..5607370)] [gbkey=CDS]
MEKVHNEQDFRAAVEKSEVTVAVFKTTWCKDCHYIEPFMPDLEKTYAGRISFIELDRDEVPDLCEELNILGIPSFIAFSR
GQELIRFVSKLRKTREEIEQFLDRAIEVGNALAK
>lcl|NZ_CP026520.1_prot_WP_042233455.1_5054 [locus_tag=PC41400_RS25445] [protein=SPFH domain-containing protein] [protein_id=WP_042233455.1] [location=5607587..5608429] [gbkey=CDS]
MTERHVWKINGFVGLLAMVVFIGAGIYSLTAELIIPAVLLFVLAGVVLSSFTVVQPNQGMVVNFFGRYSGTVRDSGFWVT
IPFSERKKMSLRVRNFNSARLKVNDVDGNPIEIAAVVVFRVVDTAKAKFEVDNYQAFVEIQSETALRHVASKYPYDSYDG
EAYSLRANSEEVAKELTLELQSRLSVAGVEVLESRLTHLAYSTEIASAMLQRQQASAIISAREKIVEGAVSMVQMAIARL
QAEGVVELDEERKAAMINNLLVAVVSDRSAQPVINSSSIY
>lcl|NZ_CP026520.1_prot_WP_042233457.1_5055 [locus_tag=PC41400_RS25450] [protein=toxin-antitoxin system HicB family antitoxin] [protein_id=WP_042233457.1] [location=5608438..5608632] [gbkey=CDS]
MAVKKNFPLRLDPKLHQALERWAADEFRSVNGHIEFLLREALVRAGRMPGHAPPPPDGESENEN
>lcl|NZ_CP026520.1_prot_WP_042233460.1_5056 [locus_tag=PC41400_RS25455] [protein=GNAT family N-acetyltransferase] [protein_id=WP_042233460.1] [location=complement(5608746..5609228)] [gbkey=CDS]
MEIRKFQVSDIGPIVSLFYETVHSVNKKDYTQEQLHAWASKEEEESRLESWKDSMSRNITYVALTDGKIVGFSDMTVNGY
VDRLYVDKDFQGQGIASALMNVLEYEAKNRGLAELETDASITAKPFFSHHGFRLVREQSVERRGIKLVNFKMKKELSRDV
>lcl|NZ_CP026520.1_prot_WP_042233462.1_5057 [locus_tag=PC41400_RS25460] [protein=copper amine oxidase N-terminal domain-containing protein] [protein_id=WP_042233462.1] [location=complement(5609373..5610947)] [gbkey=CDS]
MKVNNWKKVVCGGTGALMIVLTGCESVGGLDVNKALVNQYAVKSAESQETLKFELLTGDLAGLPPEAQKLVGLFKEVTIK
FTEEKIQDPRNASMKGEFTYGKGSIPFELSLQNEDIIIRIEGVSKPIYIRNAQTGPLNGGLLVPGQQSIQELQKVLEDKL
PTLASFFIGHAPNPKTIGVTSVSEKVNNETVSLQKLHAEIKGSEALGLVKTFLQNIAADEKGLKEFLGTVFDVFGPIAKE
AAEQSGEENKLLDAYLSNKTLAVETMYTAVQTGLKDVLASYDETTDKLVNGPGNEPIRDILSDRSSVNVDLYVDSSQQIR
KSSAGITIPLPEAETGLKGIKLSTTGERWNINQPITINPVKADGDRLTFDAADPDSAPSLGSFLKSVDKNSSIYKLLKED
LQITKRSFTLSLDDEDMSLFGFGGYRKGDVSMVPVTYVANMLDAEVKWDGETKQITVTDDLGGHTIVFTLGSAEALVDGK
PVKLEQAVELHGDYTNVPLRFLAEALGGEVKWDSEQDAVIITRQ
>lcl|NZ_CP026520.1_prot_WP_042233465.1_5058 [locus_tag=PC41400_RS25465] [protein=DUF2515 family protein] [protein_id=WP_042233465.1] [location=5611169..5612476] [gbkey=CDS]
MNGNGVKGNNPLKAIGHWLAGVPHTITELASGKLASLQSAFHLGQNAQKLVPGKDSVKRLRRTWKDLGVLPASPPAPELS
LSAQERQLVERIRLDTQRKNRNNVTRTRAYWNLYERHPELHWALLAHMVSRNGGWNMTDLEGEFLPALLKRPGREALFAF
LERANALIFGDAYPQLLLYEESKRSGEPLFHLLPAFGVSVFMRPVWEEFWAVHDPALLTVSLIVNEQHFIESRVVKQPYF
MEHVLETAEFQAQALLQLNQVFFPYALHKDPIHRSADRLNLAGEILEDFTDIKERIGFGKTLYAILFGIPAVHQGVVLFA
SAQPHTGSRADYWPHLFSRIRKDAPGVSAAGPKLEGCELKAGAAGIYSPELERTWRDVPLTAPESGDWFPAYAGSVAGYF
GDIDVPTLFEMSAECCFGLHKIQLAALASTQLKRD
>lcl|NZ_CP026520.1_prot_WP_042233467.1_5059 [locus_tag=PC41400_RS25470] [protein=polysaccharide biosynthesis protein] [protein_id=WP_042233467.1] [location=complement(5612576..5614237)] [gbkey=CDS]
MSKDSLVKGTLILTVAALIARFLGVFQRFPLVYMLGNEGMASYAIAFNLYSNLLVIATAGIPSALSKMISERTALGRMDE
ANRIYKAALIFAALAGVVMCAILFLLAPHYAESSGDPHATLATLAIAPALLVFPLIAIMRGYFQGRQRMMPNGISQVIEQ
IVRLITAVGLAYLLLGVSLDWAVAGASFGGVTGGIAALAVMLYYALKLRKEDRRTLPVSEPVYTETGQSRQSGGKLRYGQ
IYGQLLRLSIPIVIFSITVTLIYNIDSSTIIPLLRDQIGEAKAKDIVGILGGQAQSLAGLPIVLAIALSQSVVPIISGAY
SRRDLELVRNQSSRVMQLAILSGLPMVLAICTAARPLSGMFDYGGTPTGYEYGPPMIAMMTAGAMFQIIMQTSGAVLMGM
GKMRALVSSVVFGIAVKLLGNLILSPWLGIYGILASTMLCFIVMSAFNLAVMRNVVSFRIMARRRWTRLIVTAAVVVVIG
VFAERLLFEHLHLFPWRVVNATVNAVLIGGLACALYVILLFVTKVVTKDDIKTFPAPLQKLYAKAGRVLGRRG
>lcl|NZ_CP026520.1_prot_WP_042233469.1_5060 [locus_tag=PC41400_RS25475] [protein=ThuA domain-containing protein] [protein_id=WP_042233469.1] [location=complement(5614313..5615035)] [gbkey=CDS]
MIKVTVWNENVHEKTNAKVAEVYPEGIHGAIAAGIAGDSFQIRTATQEQPEHGLTQEVLDDTDVLIWWGHKAHGEVADEI
VNRVHQRVLDGMGLVVLHSGHFSKIFKKLMGTSCDLKWREAGEKERLWVVDPSHPIAEGIGQYFELEQEEMYGEHFDIPA
PDELVFVSWFEGGEVFRSGCTYHRGQGRIFYFRPGHETYPTYYNENVLKVIANGVKWAAPVRTAKPVYGNAQPLEKLQAK
>lcl|NZ_CP026520.1_prot_WP_042233471.1_5061 [locus_tag=PC41400_RS25480] [protein=DUF456 domain-containing protein] [protein_id=WP_042233471.1] [location=complement(5615122..5615598)] [gbkey=CDS]
MAILGWILIILLFAVGMLGAIFPILPGALAIFGAFFVYGLFFSFEPFGFWFWLIQSAIVIAITIADYLVSAFGVKKFGGS
KASVWGSTIGLLLGPFVIPAFGLILGPFIGAVLGELIHGSNLKHSAQVGVGSVLGFFSSVVVKVVLQLLMIILFIIWI
>lcl|NZ_CP026520.1_prot_WP_042233473.1_5062 [locus_tag=PC41400_RS25485] [protein=Cof-type HAD-IIB family hydrolase] [protein_id=WP_042233473.1] [location=complement(5615641..5616393)] [gbkey=CDS]
MSQYKLVALDMDGTLLNEEKQVSPANREAIYAALEAGVTVIFSTGRGVQSALPYAEELKLQTPIVSVNGSEVWKAPHDLL
KRTLLDLDLVRRMYDLAIEHDTWYWAYSVEGMYNRDKWAEDITKPEWLKFGFYTENKESLEIIRGELARWGELEITNSHP
DNLELNPKGISKASGIEEVCKLLGIEMSQVIAMGDSENDIAMIRAAGLGVAMGNAQDGVKRIADLVTVTNDEDGVAKIIQ
EYVLDPLSAR
>lcl|NZ_CP026520.1_prot_WP_042233475.1_5063 [gene=asnB] [locus_tag=PC41400_RS25490] [protein=asparagine synthase (glutamine-hydrolyzing)] [protein_id=WP_042233475.1] [location=complement(5616492..5618366)] [gbkey=CDS]
MCGITGVMYFEDREPSVSMLQQMTDVIHHRGPNDSGFWTENRIGLGFRRLSIIDIAEGHQPLCNEDESVWIIFNGEIYNY
KSLRSMLQDRGHVFRTNSDTEVIVHLYEEFGEECVKHLRGMFGFAIWDRRKKQLFAARDHFGIKPFYYQYNDRQLLFGSE
IKSLVASGSVSPSIRTESLMNYLTFQYVPEPNTMFEGIQKLPPAHYIKASFDGEMSIHRYWDPMFEPVDRPFGEYVEQIR
ETLKDSVVHHMVSDVERGCFLSSGIDSTAIAAHMRQIEPIRTFSVGFEGPNNETLIAAETAKALGTEHYSKIITREDFFN
TLPKAVWHQDEPVADPSAIALYHVAQLAREHVTVTLSGEGADELFGGYRIYREPLSLAPLESLPLSVRRMLHRLVKMLPA
GMKGRNYLLRGTTPLEERFLGNAKIFTEDMKAEVLRVDSEMFKRYQNPFQIAAQYYDKTKHQDPVTRMQYIDMNLWMPGD
ILMKADKMTMAHSLELRVPLLDKELFEVARRIPTKYRIAEGTTKHIFRKAMEGIVPDFILNRPKLGFPVPLRDWLKGPTG
STMVEQIKASGIEDYVKIDAIERMAKLHQNGQGDYARRLWTIYMFALWHATYMQETTKKAVAAF
>lcl|NZ_CP026520.1_prot_WP_042233477.1_5064 [locus_tag=PC41400_RS25495] [protein=penicillin-binding transpeptidase domain-containing protein] [protein_id=WP_042233477.1] [location=complement(5619014..5621059)] [gbkey=CDS]
MHNEDLEKPDSSKRTRFSVRINLFFFTTFILFSVLIVRLAILQFVEGDRLAEEENSATNRETPIPPIRGNIYDSNSYPIA
YTIPVQSVFFRIESGQNNKEEVIGLARKLADVFQQYGSKNTKPLTPEEIVELMDVGYDINQQQTKAPSYYSIPRRIKADL
TKEEVSYLMEHRDEFKWIEVMEESIRTYDQDKIAVQLTGYMRQFSTARESKYGLSFYKNKENTAEYLDTEDVGYDGLELM
YQEELRGKNGYKKYPVNSSQKIIGPVSITKPEKGNNLYLTIHKDIQNRTEQKIMDHLAYLRQPKSSSYGYSPNARSGYAV
AMEVDTGRVVAMASMPDYDPNLWTGGIKATEYKKIQPFVNNGTITTAYPDYPENELKKHANSIVYMGSTIKPLTVLAGLK
EGFFSVNDYYNDRGSYSFGKKGSQSTIWNSGRKSFGPINAADAIKHSSNTFMSAMIGERLYFDREDGLKVWDDYLKRFGL
VVSTESGLPNEYAGSNESLANAKLDSKQSALVYASWGQNEKSTTLQLAQHTAMLASRGKRMKPLFVDRITTYEGETIKKF
DPVVLSEETSFKPQYWDTIFKGMKEVSKQGFDGVNYTVAAKTGTSTQQIGSKEVDNAVFIAFAPADKPKLAVAVVVPEGG
YGSYGAAPIAREMFDAYDEYIGLNGTPRGPANGKTDGAAKH
>lcl|NZ_CP026520.1_prot_WP_042233481.1_5065 [locus_tag=PC41400_RS25500] [protein=transglutaminase-like domain-containing protein] [protein_id=WP_042233481.1] [location=complement(5621210..5622355)] [gbkey=CDS]
MFTSLFSGDTTINFISVALILIVIVSAIQGIIRGASGSAKHLLHMVFGILITIVAIYLAWQTATLVSAPVRNWLSSLAIQ
IPAQEMPVWKQTYYTAVTAIRDFELFRFGVLFLLAYLIIRQALYWFVDPVLLRWAQSRWDREASPSVISAATGGLIGSVT
GVWRALLVVAVLFVFVTLFPQAKPAGYIQASSLYQKGAQEVIQPFTGDFLQKQVPVFTRAVEQEFSQILQRKYEVLDANI
PNNISAAAQEVTKAGKTDEEKAELLYKWVGSRVEYDWNKVELYEQQRIWKEQTPEDTFATRKGVCIDYSRLYAVMARSVG
LDVKVVTGLGYDGRGGYGPHAWNEVYLADSKQWIPLDSTWFSSGGNWFNPSNFNQTHIREA
>lcl|NZ_CP026520.1_prot_WP_042233484.1_5066 [locus_tag=PC41400_RS25505] [protein=hypothetical protein] [protein_id=WP_042233484.1] [location=complement(5622400..5622822)] [gbkey=CDS]
MDAWIAFFQERWLLILVAVIVLLVVVSLVKTVVKWLLVIAIVGALVVYGANYKETLQSIGNAALTELSSQAVKAVQNEAK
DATYTANPDGTYTAATKNVKIVGTPGSSEVSVTFMNQTFKVNAEEAVRAFVEQAKKNAGQ
>lcl|NZ_CP026520.1_prot_WP_164977155.1_5067 [locus_tag=PC41400_RS29990] [protein=hypothetical protein] [protein_id=WP_164977155.1] [location=complement(5622880..5623035)] [gbkey=CDS]
MDRFIVALLAALGALAVIRFLKTRRSVRQETDRDEQIRQELKELRRKREEE
>lcl|NZ_CP026520.1_prot_WP_042233485.1_5068 [locus_tag=PC41400_RS25510] [protein=MFS transporter] [protein_id=WP_042233485.1] [location=5623177..5624457] [gbkey=CDS]
MKTAIWLYFFMFVAFFDLHAQYPILTPFALSLGAAPSFIGLIMGMYSLTHLPGNLLAGYGVDKYGSKLFIVFSLIVAGIV
LIFQSQVKDPWQLLMIRSISGFVLAFLSPACLSMLAKIAKDSVHQSKLMAGNGLIHTLASVVSPAAGAYLVAQFGFTASF
TLLGWLLIVTGLLSIPWLKETRPEAAAPPLHTGAGSAGKSVSVAMSRKPGGGGTKSAAPSPDSQGTGRKAAQLHAPVADN
YGGLAGGADKIPWMFYFVPLAISCSQGILFFELPLMKTAQTSLLTSGLFFSLISLGALVTLSMLFLNRIPPARRTVSGAL
ALAVLFFGLAVDWPLPLTVTLFMIGMAKGLIFPSLATLLAGITSSNRYGRVFSLLSIAYSIGAFIGPMVAGKLRDSISPY
FIAFVILMAALTLLPFRILRHKPAAA
>lcl|NZ_CP026520.1_prot_WP_042233487.1_5069 [locus_tag=PC41400_RS25515] [protein=toprim domain-containing protein] [protein_id=WP_042233487.1] [location=5624687..5625034] [gbkey=CDS]
MSIAIIVEGKNDRSRLRRLLSSEVNIYCTYGTPGTEALEQLKKKVGTSEVYVFTDNDSSGKKIRGRLRDLFPDGEQIYTR
RGYAGVEGTPEEYLIQQLEKAGLEEYIVYPPAPAL
>lcl|NZ_CP026520.1_prot_WP_042233489.1_5070 [locus_tag=PC41400_RS25520] [protein=SCO family protein] [protein_id=WP_042233489.1] [location=complement(5625044..5625676)] [gbkey=CDS]
MAGGFWAKNWFKVGITVVLLAIIASFAYKLYGQEQASAYLKDKPVGTAAPFELTDTEGKKVSSSDFDNKVKLVYFFFASC
PDVCPITTNFLSKVQDEMKAQKVFADKADIVSISFDPARDTPAALKEYAIRNRADFTGWKFLTGTDEQAMAQLAKKYLVG
VTKDNTGNFTHTNVILLVDKKGQIRTYYNPGRPDLDYKTIVKDIKSLSKE
>lcl|NZ_CP026520.1_prot_WP_042233491.1_5071 [gene=cyoE] [locus_tag=PC41400_RS25525] [protein=heme o synthase] [protein_id=WP_042233491.1] [location=complement(5625761..5626750)] [gbkey=CDS]
MDKQLGYKAAPDTALHADHQAEVESAVQTDVPSEPATWKDYFALTKPGIIFSNLITAFGGFWIASAGNFTIDWFFLMIFT
LVGTALVMASGTVLNNYLDRDMDAKMERTKKRATVSGKIKPSVVLGYGIILGIAGIALLFVAVNPLSSLLGLTGLFLYVW
VYTAIFKRTSVWSTFAGSFSGAVPPVIGYCAVSGKLDMTGLILFLIMFLWQPPHFWALGIRRKEEYRAAGYPLLPVVKGN
YVTKVSMMRYIVLLVPVSQLLFFYSGDAHVGPLYFFAATIMGLIWAFMGYKGFKTKDDDKWAKGMFIYSVNYLTLLFLIM
VVDSTIMNL
>lcl|NZ_CP026520.1_prot_WP_042233494.1_5072 [locus_tag=PC41400_RS25530] [protein=response regulator transcription factor] [protein_id=WP_042233494.1] [location=5627057..5627752] [gbkey=CDS]
MSSILIIEDDNSIGDMMSIYLMEDGYTVKRASDGQSGLRLFKEFEPDLIVLDIMLPDSDGMKLCADIRSLSSLPILMVSA
KNDVSDRVKALLIGADDYLCKPFSMRELSARIQALLRRSQIVKAKSAPVTREAPPSIHVDTDKRCLYVNHTPVETTFSEF
ELIKLLWLHEGKVFSREELLNRIRGIDSFVTERAIDVHIANLRKKIEVDPKDPKYIKTVWGVGYKFVLAAE
>lcl|NZ_CP026520.1_prot_WP_042233497.1_5073 [gene=gerQ] [locus_tag=PC41400_RS25535] [protein=spore coat protein GerQ] [protein_id=WP_042233497.1] [location=5627915..5628412] [gbkey=CDS]
MTISSNPGSKTYGTTAPGYTTYPSFPGTAQGGYPSSPGSQFPGMPFPGSQLPGYPSYPGQYPPPGAGSGGTGVPGGVSQL
PLEESFIENILRLNLGKVGTFYMTYENNKEWNAKIFRGVIEAAGRDHIIISDPKTGVRYLLLMVNLDYVTFDEPLNYFNK
PVQPR
>lcl|NZ_CP026520.1_prot_WP_042233499.1_5074 [locus_tag=PC41400_RS25540] [protein=hypothetical protein] [protein_id=WP_042233499.1] [location=complement(5628547..5628840)] [gbkey=CDS]
MSKVFVEYAILPESREVYLSFMRERLRDNEKLSLLEGTDQPNLFVEIWTPVSPDEYTRMREVRLGSGDPIWGRMHPWIKG
GREKLHIWHFRSIGPDE
>lcl|NZ_CP026520.1_prot_WP_042233501.1_5075 [locus_tag=PC41400_RS25545] [protein=metal-dependent hydrolase] [protein_id=WP_042233501.1] [location=5628929..5629912] [gbkey=CDS]
MDTGTHLVIGFGLAGLSMVDPVVASNPSVMTAVFIGTVLGSQAPDFDTLLRFKNNAVYIRNHRGISHSLPFVALWTLLIT
GILALIFRGVPVLHLGMWVLIAVAFHVFTDLFNTYGTQAFRPFNQKWISWNIIHIFDPFIFTAHLVAIFMWAVHAAEPTL
IFPVLYGLIALYYVWRTLAHYLVEKKLPVQDPVHKAGDKYMAIPTYSLTAWNVVKQGEDGLYTLGDLRKGNLRWIDQVKC
DIHPAAEASKSHKDIQAFLYFSGHACAQVIEKPWGYEVRWSDVRYRHRKQYPFVAVLLMDHSYKPLDSYIGWLSDDRLQK
RLRVDSY
>lcl|NZ_CP026520.1_prot_WP_042233503.1_5076 [locus_tag=PC41400_RS25550] [protein=alpha/beta-type small acid-soluble spore protein] [protein_id=WP_042233503.1] [location=complement(5630076..5630300)] [gbkey=CDS]
MGTGQSRSSNVLVVPQANQALEQLKYEVAQELGIQIPQDGYYGYMATRDTGAIGGHITRRLVQIAEQQLAGQGR
>lcl|NZ_CP026520.1_prot_WP_009673090.1_5077 [locus_tag=PC41400_RS25555] [protein=DUF5325 family protein] [protein_id=WP_009673090.1] [location=5630429..5630620] [gbkey=CDS]
MSRGLALLFAVIGTLLLAGISFAIALREPLLVVLFIVLSIGFMGFGFATKAKMRRKREASGKQ
>lcl|NZ_CP026520.1_prot_WP_042233505.1_5078 [gene=trpS] [locus_tag=PC41400_RS25560] [protein=tryptophan--tRNA ligase] [protein_id=WP_042233505.1] [location=complement(5630686..5631669)] [gbkey=CDS]
MKRVLSGIQPSGQLTLGNYIGALRNFVKLQNEHQCYFMIVDLHAVTVPQEPAALKEQTESVAALYMACGIDPAKASVFLQ
STVKEHAELGWLLTTLTYIGELERMTQFKDKSAGKESVGAGLFTYPSLMAADILLYNADLVPVGEDQKQHLELTRDLAAR
FNHRYGDTFTIPEPFIPEVGARIMSLDDASKKMSKSNPNPGSFIAMLDEPSVIRKKLSRAVTDSGREVKFDPKNKPEVSN
LMGIYSQCADMTVAEVEARYEGQGYGAFKKDLAEVVVAALEPIQQRYREIRESGELYDVLRKGAAEASELAGETIKQVKE
RMGFVLL
>lcl|NZ_CP026520.1_prot_WP_042233509.1_5079 [locus_tag=PC41400_RS25565] [protein=nitroreductase] [protein_id=WP_042233509.1] [location=complement(5632101..5632673)] [gbkey=CDS]
MTTAAASILGLLNKRRAVRGYTEQPVEEQDIRTLLQAAVTAPNDRLREPWHFYVIQGEAKVRYEAAAEAFLTERFPTKPH
LVEETMKAVKGVPLVIVVTADVIPGDDPSSRDNEYAAACAIYGMWLTAEEMGLGLVWRTRGVGLVHDGRMHEFIGSPENR
KVVGTLFIGHPAEPAKPTSRTPYEEKTTWL
>lcl|NZ_CP026520.1_prot_WP_042233512.1_5080 [locus_tag=PC41400_RS25570] [protein=cobyrinate a,c-diamide synthase] [protein_id=WP_042233512.1] [location=complement(5632826..5634406)] [gbkey=CDS]
MAAEQEKEPGTLSGTAETSAADFVPEAGGPIVSRAKRLLIAGTGSGVGKTTVTIGVMAALKARGLSVQGFKCGPDYIDPT
YHTAVTGRLSRNLDSWMLPADTVREIYCRASRDADIAVIEGVMGFYDGKNPKTNEGSSAEISLLLQCPVILVVNCKSMAR
SAAAIVKGFQLLDPRVRIAGVFANLVGSEGHYRIVRDAIEQECGVPVIGYMLRDENLAMPERHLGLVPSIGRGELSPFFS
QLAERISAHTDLERLLQAADTEPLEASARIFRPLEADGGGTAPEAPRSPLIGAHANADAAHEKVRPPEPRVRIAVAKDAA
FHFYYPENLELLQAYGAELVYFSPLAGEKLPAGVDGLYLGGGFPEEFAERLAGLQEMKQSIAEGIVSGLPTLAECGGYMF
LTDAIVDTSGRQYPMLGLVPGKVTMQTKLAALGYREATGTEGNFLLGPGDTARGHEFHYSVFEPAASATLPPAYKTKGMR
GSKEEGALKGSLVSGYTHLHFASNPDMPRRWIEKCLLFRKMREGLV
>lcl|NZ_CP026520.1_prot_WP_042233514.1_5081 [locus_tag=PC41400_RS25575] [protein=cobalamin biosynthesis protein] [protein_id=WP_042233514.1] [location=complement(5634437..5635678)] [gbkey=CDS]
MISLQEGIVPEIRVRSDYAVIAITKHGVELARSLLRSFGGVDVYYMSKFARGDEDVVFNGGVEGLTSASTTASGAAPGEA
AAPNTASPYGGIQLFEGSVRMLFPALFPAYKGIICIISLGAVVRMIAPLLQDKKKDPGIVVIDDKGEHVISVLSGHLGGA
NELAREVAAAMGARAVITTASDVQKTIAVDLFGRRFGWEWESADKLTPVSASVVNEEPVAVVIESGEPGWWTLDTPVPPN
ICSYGSVQEALEAAPHAALVVTHRLLSSAEQTILNNGVLYRPKVIVLGIGCNRGTPAEEIEEVIASTLAELDFSIRSVKT
VCTINLKADEEGLLAVCGKYGWPLVTYTPEELNEMPMEQPSPTVFKYTGAYGVSEPAVQRYIGGSVPLALVKKKSGNVTI
SVGIQPFTEELPQ
>lcl|NZ_CP026520.1_prot_WP_009673080.1_5082 [gene=cobM] [locus_tag=PC41400_RS25580] [protein=precorrin-4 C(11)-methyltransferase] [protein_id=WP_009673080.1] [location=complement(5635675..5636451)] [gbkey=CDS]
MSRVVIIGAGPGDPDLITVKGLRLLQEADVVLYTDSLVNDDLIARAKPEAEVLKSAGMELDEMVALMIKRVEEGKLVARV
HTGDPAVYGAIMEQMTLLKKAGVPVEIIPGVSSVFGAAAALQAELTIPDLTQTLILTRAEGRTPVPEAEQLRELAKHHCT
LALYLSATLVKKVVGDLLEAGWSKDTPVAVVQKATWPDELIVRTTVENLEADMRSNGIRSHAMILAGWALDPHIHDKEYR
SKLYDKSFTHRYRRGVKP
>lcl|NZ_CP026520.1_prot_WP_042233516.1_5083 [gene=cobI] [locus_tag=PC41400_RS25585] [protein=precorrin-2 C(20)-methyltransferase] [protein_id=WP_042233516.1] [location=complement(5636448..5637224)] [gbkey=CDS]
MSEYAANGTRSGGAGQPANRAESSATGTLYGLGVGPGDPELLTVKAFRILKQSPVIAYPRKKSGAQSYALKIAETYVDTT
AKEMLGLTFPMTKDREALEKSWNKTVDAVWEHLAAGRDVAFLTEGDPMLYSTFIHMMRFLKQRHPEAKTVSVPGISSVNA
SASRLGIPLADGDERIAIVPATDDYGAVREAIEEHDCVVFIKVAKVIDLMLQVLGDLGLTDKAAVLTKVTSAEESVWLDV
RELQGRQLEYLTLMVVRK
>lcl|NZ_CP026520.1_prot_WP_042233519.1_5084 [locus_tag=PC41400_RS25590] [protein=bifunctional cobalt-precorrin-7 (C(5))-methyltransferase/cobalt-precorrin-6B (C(15))-methyltransferase] [protein_id=WP_042233519.1] [location=complement(5637221..5638477)] [gbkey=CDS]
MNKPIKVIGIGDNGAESLLPAYRQWIEESEVLVGGERHLAFFPDYRGEKKVLKGGLSAMTDGLAAEERTVVVLASGDPLF
YGIGSLLARKLTVELYPNLSSLQLAFARMGESWQDAHVVSLHGRSIKGLAQRIDGRSKVALLTDEHNTPAAIAAYLLEYG
MTEYRAFVAENAGGAAERTGWYELEQLARTECAALNLVVLKRTGPSPVWPLGIPDEQFVQRKPDKGLITKREVRVLSLAQ
LALRPDSVVWDIGTCTGSVAIEAGRIAREGAVFAIEKNEGDLANCHENARRFRVDLTAVHGRAPEGLEAFPDPDAIFIGG
TGGEMRELLRICALRLKQGGRIVLNAVTIENLSDAMRAFAEEGMRTDIQLVQISRSKPILQLTRFDALNPVYILTAVHPP
QENTDTFDTPVEKGDTAT
>lcl|NZ_CP026520.1_prot_WP_042233521.1_5085 [locus_tag=PC41400_RS25595] [protein=cobalt-precorrin-5B (C(1))-methyltransferase] [protein_id=WP_042233521.1] [location=complement(5638474..5639610)] [gbkey=CDS]
MEEQGAKPLRHGYTTGSCAAATAKAALTALIEQELQNEATIRLPIGEIVTFTMTACSVGEESASAETVKDGGDDPDATHG
ARIITEVKWRSEPGIVIDGGIGVGRVTKPGLPVPVGEAAINPVPRKMIREAVEDVLNVYGSDRGVQVVVSVPDGEEIAKK
TLNGRLGIIGGISILGTRGTVVPFSTSAYKASVAQAIRVAVRCGCEHIVLSTGGRSEKFGIDLYPGLPEEAFVEMGDFVG
FALKQCRGQGVKKITLVGMMGKFSKLAQGVMMVHSKSAPVDFGFLAELAESAGADAELIGEVLSANTASQVGDRMQELGL
TAFFEEMSSRCCRESLKEMGMEQAFGENDGAVTIETFLISMKAQLLGQAALYGKGKDK
>lcl|NZ_CP026520.1_prot_WP_042233523.1_5086 [locus_tag=PC41400_RS25600] [protein=precorrin-8X methylmutase] [protein_id=WP_042233523.1] [location=complement(5639621..5640271)] [gbkey=CDS]
MDFRTEFKPLTVQPQEIEAKSFEMITEELGPHDFTEEQYPVVQRVIHASADFELGRSLVFHPRAIEAGIEAIRAGRTVVA
DVQMVQVGISKPRLEKFGGKVNVYISDPDVMEEAKRLNTTRAIISVRKAIREAEGGIFAIGNAPTALLELIRLVKEGEAR
PGLIVGVPVGFVSAAESKEELLQLDVPFITNIGRKGGSPVAVAAVNALSLMAEKRG
>lcl|NZ_CP026520.1_prot_WP_042233526.1_5087 [gene=cobK] [locus_tag=PC41400_RS25605] [protein=precorrin-6A reductase] [protein_id=WP_042233526.1] [location=complement(5640271..5641050)] [gbkey=CDS]
MILLLAGTSDARELAVLLAQAGHRVLATVVTDSAAKSLADAGIAAAVGRLDAEALAALIRREEVLLVVDASHPFAEEASR
NAMAASKAAAVPYVRYEREGSDLQVRPGIVFADDYAHAAELAAVRGGVVMLTTGSKTLHVFARRLGGLPDVRLVARMLPR
LDNMEKCEELGIEQRNIVAMQGPFSKELNKALYAHYGVTLMITKESGKIGAVDEKLEAALEMGIETIVIGRPGLEYGIHF
SDYEGVLKAAADLLQEVSS
>lcl|NZ_CP026520.1_prot_WP_042233528.1_5088 [locus_tag=PC41400_RS25610] [protein=sirohydrochlorin chelatase] [protein_id=WP_042233528.1] [location=complement(5641047..5642243)] [gbkey=CDS]
MDAVLFVGHGSRDAEGNSEVRAFVSRLAETMEQGIIETCFLELEKPTISQGIQTCVNRGATRIAVVPIILFSAGHAKIHI
PGAIDEAKERYPGVQFIYGRPIGVHEGVVDIMTARLGGADRFREETPGDTAVLVVGRGSSDPDANSELFKISRLLWEKLQ
VPWVETAFIGVTAPLLENGIERCLRLGAKKVIVLPYFLFTGILIKRMEGMLLDFSLRYPDCTFEMTDYFGLDPILGGILR
DRAGEALQDEVKMNCDMCSFRLEAMKHIDHHHHHDHDHDHHHHHDHDHDHHHDHDHDHGHEHEHVHGHAHDHDHAHHDHS
HEHHEHANGEEHHGHNHTHGEQPHTHGHVHDHGEHEAHAHNHADNRRAGDEAGASGGEAAASAPGPVAAGADGREGGR
>lcl|NZ_CP026520.1_prot_WP_042233676.1_5089 [gene=cobJ] [locus_tag=PC41400_RS25615] [protein=precorrin-3B C(17)-methyltransferase] [protein_id=WP_042233676.1] [location=complement(5642317..5644233)] [gbkey=CDS]
MAGKLLIIGFGPGDEAHITKRAREAISESEVIIGYNTYVDLIRSLIDGKQVVRTGMTEEVSRAQEAVRQAEAGRKVAVIS
SGDAGVYGMAGLVYEVLIEKGWKPSDGVEVEIVPGISAINSCASLLGAPVMHDACTISLSDHLTPWDLIRRRVEAAASAD
FVIALYNPRSGRRTRQIVETQQILLKHRSPDTPVGIVKSAYRDREHVVLTTLAEMLEHDIGMLTTVIIGNSATRNYEGLM
ITPRGYQRKYTLDSDIQPLRPHERLRREAEPWALEQDEETGAEAAPALGEQSGAGMRSAEEREAGSAVLPSFFDEDEYED
DGYGAYSPEAEGAGTTDDYDREDSTVRIPAAGTRSAYDTPASSGAGEYGDSAVAVAVRPPAAPAQAAASAPTAFELAEEA
LRLVTGEPGGPAASASPGGFFRQQAILEFAVSPGLANKNLTAKQLMVLADAVGDTGTMEYTPHHQLLVKLRTDDPDSVTK
RLRGEGLLLAPVGDVVQIKACDFCNMDKGDAVPYAEELQDLLGGAAVPKEMKIGFNGCGMACYGAVKEDIGIVYRREKFD
LFLGGKTVGRNAHAAQPVAEGIEPDKLVETIERIVARYRNEGHPNERFHKFFARVKELEGYRHREVPAFQIENALCGD
>lcl|NZ_CP026520.1_prot_WP_042233531.1_5090 [locus_tag=PC41400_RS25620] [protein=alpha-glucosidase] [protein_id=WP_042233531.1] [location=complement(5644872..5646560)] [gbkey=CDS]
MNKTWWKESVVYQIYPSSFKDSDGDGRGDLRGIISKLDYLAELGVNVIWLCPVYLSPGADNGYDISDYYAIDPQYGTMED
FDELLLEARKRGLKIMMDLVLNHTSDRHAWFKESRSSRDNPKRDYYIWRKGKEGLFPNNWESYFSGSVWEHDTETDEYYM
HLYSKHQPDLNWENEEMVEELFRMVHWWLGKGVDGFRFDAIAHIVKAQGLPDADNPQHLPVVQAYQLFSNLEKVHTLLHK
LNDRVLFNYDIMTVGETSGLGPEQALDYVGDERHELNMVFQFEHMNLDAASPGTGKWVSKPWTLLELKNVMSRWQTVLHD
KGWNANYLGNHDQPRPVSRFGNDGPFRVESAKMLATFLLTLEGTPYIFQGEEIGMTNVAFDLIDDYRDVETINYYKQAKL
LGKPQDEVMKAIWKKSRDNARTPMQWDDTANAGFTTGEPWIKVNPNYPGINAAAATADPQSIYHYYRSLIALRKKHPVIV
YGEYKLLLPLDTEIYAYTRALNGEKLLVILNFFDGTPEFTWPEEPGRGKDAELLISNYDPVPGEDTGVIKLRPYEARVYL
LG
>lcl|NZ_CP026520.1_prot_WP_042233533.1_5091 [locus_tag=PC41400_RS25625] [protein=glucose-1-dehydrogenase] [protein_id=WP_042233533.1] [location=complement(5646580..5647365)] [gbkey=CDS]
MYKDLEGKTVVITGASTGIGRAAAVRFGQEKANVLINYLSHEENIPALIAEVEAAGGRALALRGDVTKEEDVKKLVETAH
EHFGSLDVMINNAGIENEVPSQELSADDWRKVLDVNLTGAFLGSREAISYMLEHEIRGAVINVSSVHEMIPWPHFVHYAA
SKGGIKMMTETLALEFAPKGIRVNNIGPGAIDTPINESKFADEQARAGVEALIPMGYIGKPEEIAAVAAWLASSESSYVT
GITLFADGGMTKYPSFQGGRG
>lcl|NZ_CP026520.1_prot_WP_164977156.1_5092 [locus_tag=PC41400_RS29995] [protein=hypothetical protein] [protein_id=WP_164977156.1] [location=complement(5647540..5647695)] [gbkey=CDS]
MGYVGGVGGVGGVGGVGGVGGVGGYGGIGTSVGAILVLFILLVIISRTFLY
>lcl|NZ_CP026520.1_prot_WP_042233535.1_5093 [locus_tag=PC41400_RS25630] [protein=metallophosphoesterase] [protein_id=WP_042233535.1] [location=complement(5647834..5648697)] [gbkey=CDS]
MDLNPKMTRRAFLAKGKFLIGALLAAPFVSYGYARFAEPHWIRTKHVRLELERLPQAFDGVRVVQFSDVHVGPYLAPGDL
PQLVDMINALKPDLLCFTGDLYDYRVYDGSLVSQALAALKAPLGKFAVLGNHDYYGSPVETEKVLKPGGFELLTNRSVAV
GKDKALIRVAGVDDMWEGRPDLDRALKSVRRDDFVLLLSHAPDFADIALEAPVDMQLSGHSHGGQVRLPFYGAITTPMYG
RKYVDGLYKLGGGKLHVYTNRGIGMTMHPVRFWCRPELTVFTLKKRL
>lcl|NZ_CP026520.1_prot_WP_042233538.1_5094 [locus_tag=PC41400_RS25635] [protein=GntR family transcriptional regulator] [protein_id=WP_042233538.1] [location=5648951..5649328] [gbkey=CDS]
MIKINERSELPIYEQIIQGMKELILKGALQQGDRLPSVRELSAAVLVNPNTVGKAYQELERQGIIETLRGKGTFVAKPSS
PGMEKERLEKFRSTFRQLLVEAAYLGIKADQVKDWTEQFYKEIKE
>lcl|NZ_CP026520.1_prot_WP_042233539.1_5095 [locus_tag=PC41400_RS25640] [protein=ABC transporter ATP-binding protein] [protein_id=WP_042233539.1] [location=5649342..5650232] [gbkey=CDS]
MLNVEALSKRIDGRPVLEDVRFRLGQGSIAALVGRNGAGKTTLLRTLAGILEPDRGEVQWEGRSIHEHPDAKRNIVYVPD
SLDMLGGYTAKECADLYRLAYPDFDPAYFADQLRRFHLPERKKIRTFSKGMKALIGLLLAFSSKASLILLDEPTNGLDSI
VRKQVLSFLVEEVASGNTSVLISSHQLQELERIADTVIMIREGKLESASSLEETKGALHKLQVVFNGEPPGDFLSLPQVR
VVSHVGRVYTLLLLQDDSVQTLQHLQHLNPLLVDRLSIQLEDVFEYRLGEGDEHAV
>lcl|NZ_CP026520.1_prot_WP_241688737.1_5096 [locus_tag=PC41400_RS25645] [protein=hypothetical protein] [protein_id=WP_241688737.1] [location=5650192..5651256] [gbkey=CDS]
MSTVWERVTNMLSKALWKKDYKQAKLLLWLMPAAGFLFLVFNRINELILSPANMKWKLQFINKDSENATIYSFGSEFTNY
RIALAVLALGVAVLLIGMERRGLHQDFTFSLPFKRTQIYLNKWLLGILFILGAVLLTNALDIVVILLSPYHDYFTLSVHV
REVLRSTFILTAWYTIVLFAGTFTGGALTQILLSLVAGVLPMGLRSMLGSFLSVYWPGYRELLNPVIRHNWHYLDPLDLV
TGGGYSLTSLYLFGICAVYILIFLGAGLYAYSKNKPENNGKAILFPGLEPYFLWGAVVCGACFAGMVGKNFTAPYYQYQQ
NGASLYAYGLAFLLGGTIVYFVLRFLLKRRFKFI
>lcl|NZ_CP026520.1_prot_WP_042233543.1_5097 [locus_tag=PC41400_RS25650] [protein=AI-2E family transporter] [protein_id=WP_042233543.1] [location=complement(5651935..5652969)] [gbkey=CDS]
MFTKQYFQSKGFRRILIFALLALILYAMQSMINLILLTFIFAFLMDRMQSSVSSRLNRYMRVDRRWIVAILYVVILALLG
IGMYRYLPVVAVQVSQVVRQLYEFYTQPQDNEVLSYIINLIKPYIEQTELSKYMNQGFGFVFKSVTDISKVLVQVFIALL
LSLFFLLEKPRIIRFTAKFKDSKIAAFYEEVAYFGSRFVHSFGKVLEAQFLIALVNGILSVIALWILGFPQLFGLGLMIF
LLGLVPVAGVIISLIPLCTIAYSIDGIMMVVYVLVLVAILHALEAYMLNPKLMSAKTNLPVFYTFIVLIFSEHFMGVWGL
IIGIPIFMFLLDVLDVTREKEPAA
>lcl|NZ_CP026520.1_prot_WP_042233545.1_5098 [locus_tag=PC41400_RS25655] [protein=hypothetical protein] [protein_id=WP_042233545.1] [location=5653572..5653991] [gbkey=CDS]
MKKIIVLTLLFLLTLGVSSAYANTETEPNDTYTTANVLSQGDNFGKLSGSLDRDWWTDQVPAGSVHKVTLTSPTGYPYLI
TLYTLSPLGVPIYQSSASGNGSVTLNGGASGTTFFVKVTPAFLNVDTRKNYTLNLSIVN
>lcl|NZ_CP026520.1_prot_WP_042233549.1_5099 [locus_tag=PC41400_RS25660] [protein=ABC transporter permease] [protein_id=WP_042233549.1] [location=complement(5654083..5655069)] [gbkey=CDS]
MKDKALDVSFRYGTLVVIAAVFVFFSITNPNFLSYDNIFDILRSISIVTFVAIGVTLSQIVDGFDLSVGSTVSITTVVAA
SLMVWYEQPLIVVLLVPLAIGIGIGLLNAFLIVKVRIPDLLATLTVMYIIAGVHKTYTKGFSVYNNMPMPDGTTAKGVVT
ESFKWLGQGKIIGLPVPVILMIIAVAAVHIFLTYTRWGRQLYITGGNREAARLSGIGVNRVRTAAYVLSGLFAAIGGILY
AARIGSGQIDAGAPLLMEAVAAVFVGYSVLGAGKPNVIGTFFGAVLIGILLNGLTMLNVPYYSFDIIKGGVLLAALAVTF
VHLNRRRQ
>lcl|NZ_CP026520.1_prot_WP_042233550.1_5100 [locus_tag=PC41400_RS25665] [protein=sugar ABC transporter ATP-binding protein] [protein_id=WP_042233550.1] [location=complement(5655059..5656579)] [gbkey=CDS]
MTTTSLSVLRMNGITKTFPGVKALSEVDFEVKGGEVHGLLGANGAGKSTLMKILSGAYTADGGTIEIDGKPVPISSPKDA
MRQGIFCVYQEVDTALVPELTVAENVLLDSVAHGSAWLRWSTLNREAARILDSIGSDIPVKLKASQLTIAQKQLVLIARA
VAHAAKCVIFDEPTAPLSLEESEQLFAIMERLKANGVGCVFISHRLAEVTRLCDRITVMRDGSRVLTKAAQGTSHGEIIE
TMLGKTFAEEFPKLPADLGGTVLEVRGLRRGTKVRGVDFTLREGEILGIVGLVGAGKTELSRLLFGADRADGGDVLLRGK
RLRLHEPEDAVRAGVVLVPEERRKQGVLVEESVQRNLSLPILRRLSAGGFIRRREERGHAERLIGRLGVKPPQPERPVKH
LSGGNQQKVAVGKWLPTDASVYMFDEPTKGVDVGAKSDIFRLIGQLAQEGKGVIYLSCEIAEILGIADRILVMSYGRITG
ELSREEATAEAILSYASAGEAEHVER
>lcl|NZ_CP026520.1_prot_WP_042233552.1_5101 [locus_tag=PC41400_RS25670] [protein=sugar ABC transporter substrate-binding protein] [protein_id=WP_042233552.1] [location=complement(5656652..5657764)] [gbkey=CDS]
MKGIRKVVHVFTMMILIVGLVAACGKANTETTTGSAAQTPGASPAGGDKANALAGKKVALIMRQNLGTFSAQYVEGVKEQ
VTKLGGTVTVIPADGDLTKMASNLDAAVNQKFDGILLDHGTPDALQSGINKAVEKKIPIVAFDSNVKGPGIVALEQNDQK
LAELTLEQLGKDAGGKGNIVKVWVAGFPPMESRQIAYKAFLDKNKDIKEIAAFGNAQNPALDTQAQMEAILKKYPNKGDI
TAVWAAWDEFAKGATRAIQQAGRTEIKVYGIDMSDEDLQLIQDANSSWVASAAVDPKDIGRIQVRYLYQKLHGDQTPEKV
VLEPVFVSRDQLPKDKKVTTAELSQHVQGWGNSQQGYTDWLKEAEKQSGK
>lcl|NZ_CP026520.1_prot_WP_042233554.1_5102 [locus_tag=PC41400_RS25675] [protein=viroplasmin family protein] [protein_id=WP_042233554.1] [location=complement(5657971..5658612)] [gbkey=CDS]
MAEKKFYVVWVGRTPGIYSTWAQCKEQTDAFPQAKYKSYPTEAEARAAYERGWQKSLDFGAKKKSAGAGSSGKSEAPPEV
DYDSISVDVGCSGNPGIVEYKGVDTKTGEVIFYRGPISKGTNNMGEFLAIVHALAYLQKQGSSKTIYTDSMTALSWLRKK
EVASNLVRDASTKEIWTLVDRALEWIRTNTYTNKIVKWDTKKWGEIKADFGRK
>lcl|NZ_CP026520.1_prot_WP_009673104.1_5103 [locus_tag=PC41400_RS25680] [protein=alpha/beta-type small acid-soluble spore protein] [protein_id=WP_009673104.1] [location=5658905..5659132] [gbkey=CDS]
MGAGQSRSSNTLVVPQANQALDQLKYEVAQELGIQIPQDGYYGYMATRDTGAIGGHITRRLVQIAEQQLAGKLGR
>lcl|NZ_CP026520.1_prot_WP_042233556.1_5104 [locus_tag=PC41400_RS25685] [protein=hypothetical protein] [protein_id=WP_042233556.1] [location=complement(5659211..5659573)] [gbkey=CDS]
MIIEPISLVRQVEIVFTELRQELSELNSGTVFLQIRNNAVGKFGVRHNPLEGRDGVVQKSGTGLTESQWQAFRKLALDSL
KYKVNWTHGEITFDFTVKKETLYASVMFESNYNMSSLIRG
>lcl|NZ_CP026520.1_prot_WP_042233559.1_5105 [locus_tag=PC41400_RS25690] [protein=HAMP domain-containing methyl-accepting chemotaxis protein] [protein_id=WP_042233559.1] [location=5659925..5661184] [gbkey=CDS]
MKNWSSLSFKSKLLAGCYGLIAVYSAALLLLLLLPGFNRLIGFIFLAVMLAASFPFVRWFERALTEPVEDLTRAALNISK
GDFSRKVDATTDDALGDLGRSFNSMIDKLRGILGETGSITRQVVDTSQSIYSQNLQLKDVLQQVTVSAQELASGAGQISE
EVSTISVSTKDIEQKVSNFAGSTQTMNARSEQMLSLVDKGREAVEIQGQGMKRNVEATARVSDTIRTLAEQTAGISKITK
AISEIAEQTNLLSLNASIEAARAGEQGRGFAVVAQEVRKLAEEATTQTREVFGLVTAIESGIRQALSHIEVNEEAVGRQT
ELIQDTEAVFSDIFNAVRFISDRIGEFAEESDSMLAGARQISATMENISAITEQSAAGTEEVSASMNEQISGVLEMVSRC
EQMTRSVKQLEQTIQIFKL
>lcl|NZ_CP026520.1_prot_WP_042233561.1_5106 [locus_tag=PC41400_RS25695] [protein=M3 family oligoendopeptidase] [protein_id=WP_042233561.1] [location=complement(5661281..5663083)] [gbkey=CDS]
MHTPLNQTWDLDVFFPGGSESPQFASFLNDLDGDIAGFRASLEKASSPQSAGDSAGFKALIDQYQGILKRIREADSFVGC
LAADNQKDKKAVQLGGKVKTLYADFLASLTRLDPVITGTPDDVWSALLEQEGLREIAFSLNERRTQAKEKLPPEQEALIS
DLAVDGYHGWNDLYNTTVSQFRMVVEENGERQELSAGQAFNRLHTPELAKRTELFNRWEEAWAEKADFAAEALNHIAGFR
LQVYKQRGWDSVHKEPLEINRMSKQTLDVMWDVITRNKGVLVDYLNRKAQLLGLEKLAWADVDAPLEGGTKTVGYDEGAK
LIVDQFRDFSPRLADYAAQAFENRWIEAEDRAGKRPGGFCTSFPLAKQTRIFMTYSGTLNNLSTLAHELGHGFHQYLMDD
MPALAQEYAMNVAETASTFAELIVSDAAVKAAVSDDERIVLLEDKIQRAVAFFMNIHARFIFETNFYDERRKGLVSADRL
SELMVQAQKDAFNDSLSAYHPHFWAAKLHFYATDVPFYNFPYTFGFLFSAGIYARALEEGTAFEDKYAALLRDTGSMTVE
ELAHKHLGVDLTQPDFWQSAVDTVIEDVNEFMRLTEGKVK
>lcl|NZ_CP026520.1_prot_WP_084706668.1_5107 [locus_tag=PC41400_RS25700] [protein=YycC family protein] [protein_id=WP_084706668.1] [location=5663312..5663488] [gbkey=CDS]
MKPLNISPDTAVKLAKHLGVPLEHLMHMPQHILLQKLAELASSKPDKDPAPDSESETK
>lcl|NZ_CP026520.1_prot_WP_231392066.1_5108 [locus_tag=PC41400_RS30460] [protein=hypothetical protein] [protein_id=WP_231392066.1] [location=5663493..5663729] [gbkey=CDS]
MLPFSHTWPYETVLKDIYVAECPFCSQSNVLLPLRTKDLLPIHEGKKRLLVFPCCREKVTLVDTDTDYLLADRPFRKR
>lcl|NZ_CP026520.1_prot_WP_042233563.1_5109 [locus_tag=PC41400_RS25710] [protein=GGDEF domain-containing protein] [protein_id=WP_042233563.1] [location=5663768..5664883] [gbkey=CDS]
MSEYLVALTIACTLVTLSYLALKIQRMHFFENYEKIASPVLTGLACILLMQQPLSLDTGAVDLRYLPVVMAGLRYGLGNA
LFSTLLPTIYSIVTTEDSMWMMQTLQSLIIPAVLSSGFHRSEFRTGSKPIPIRAGLKISVLLFLTQVVIYAAVHTGGHAL
SESYPLLFMSVVSAVSVCLLIGMSNDENAKWLNQRKLEMKASQDLLTGLPNLRSFFEIAGRTLGFRAVSIFMIDIDNFKQ
YNDCWGHLHGDQLLRETGRLLKNVVAEHDYVARYGGEEFILMSTETDPVKLREYAEKIRLAVAAHPFEHAETTPVTVSIG
ISSAADAAADLQLLIDEADRALYISKNTGKNRITLCSESPEPALLEQQNAY
>lcl|NZ_CP026520.1_prot_WP_042233566.1_5110 [locus_tag=PC41400_RS25715] [protein=globin] [protein_id=WP_042233566.1] [location=complement(5664976..5665362)] [gbkey=CDS]
MNIDDSTTTLYERMGGEPVLRRLVEAFYPKVQENPLLGPLFPEDIRPVMEKQIQFLTQFFGGPMLYTMEHGHPMMRARHM
PFPVTRERADAWLSCMREAAREIGLPDDLAGILLARLQAPAYHFVNQE
>lcl|NZ_CP026520.1_prot_WP_042233568.1_5111 [gene=ylbJ] [locus_tag=PC41400_RS25720] [protein=sporulation integral membrane protein YlbJ] [protein_id=WP_042233568.1] [location=5665706..5666932] [gbkey=CDS]
MSRRSVFTLIPLCVLIIGIAALMVAYPRESLAAAVHGVAIWWGVLFPALFPFFVISEVLLGFGIVHFFGMIFDPMMRPVF
RIPGIGGFVVAMGFASGYPVGAKLTSQLWKDRLVNREEGERLVAFTTSSDPIFLIGAVCVGFFHDASLAAVLAAAHYGGA
LLVGFLMRFHGRGAPSTPAPRAAAGNRSSVWARAFKAMHKARLEDGRPFGTLLAQGVRNSLNLIFLLGSLVVFFSVVLEM
LTMSGVIHMLSGLLRMFLNVLQLPPALSDALIGGTFEVTLGAKAAGGMDDSTLLVFKTAAAAFILSWGGLSVHAQIVSVL
SQTNMRYAPFLTARILHSLLASVLVFVLWKPMRPIGFLLENALPAGKFGTDGATGWLTLFPLAFVVFAGSLFILFSLYLL
HSFLKRTV
>lcl|NZ_CP026520.1_prot_WP_241688738.1_5112 [locus_tag=PC41400_RS25725] [protein=PilZ domain-containing protein] [protein_id=WP_241688738.1] [location=5667218..5667898] [gbkey=CDS]
MMNWTYALTQHILVPKTECKLIIVGENADGQPFCLETRVFLREAGLDFFSVHVYFDLANPLDKLEQISFIEFSFQDQGIN
YYSFVDLVQLDPEVSSCVLKLVSPQELYSYQNRRHIRKPLPPLTNATCRVIGARQNSAEDTPAFIGQLLEISRSGTSMIT
SHRMIESLMLELSFHLPEVEAPLLLTGEVRSSTPFSSDSYRVGIEFHDVPLSSLNLIDTYCEQTSS
>lcl|NZ_CP026520.1_prot_WP_042233573.1_5113 [locus_tag=PC41400_RS25730] [protein=NAD kinase] [protein_id=WP_042233573.1] [location=5667910..5668722] [gbkey=CDS]
MKYFLVQRQDDVSAGLARRFHELAGQYGMELDEQEPEVVVSIGGDGTMLIAFHQFADKLDRISFVGIHTGHLGFFADWKP
DQLDYLVELMHNGEINQDMRIVQYPILEIEIHSSQGVEQRLALNEFTLKGIENTLVAQVDINDEMFEMFRGDGICVSTPS
GSTAYNKSLGGAIVHPSLAAIQIAEIASINNRVYRTLGSPMLLPMHHHCDIYPKSKHPVYLAFDYQYVQRSDVESIRCRV
SPNLKVKFARYRPFPFWNRVREAFIGHEFQ
>lcl|NZ_CP026520.1_prot_WP_042233575.1_5114 [locus_tag=PC41400_RS25735] [protein=type II CAAX endopeptidase family protein] [protein_id=WP_042233575.1] [location=5669234..5670097] [gbkey=CDS]
MKTNFGSTSVRGSEGEKERICFARKGLLVFFAMLIPLSVLGYVLTMKYPIFVLFLMWTPGLSSIFTRLLLREGIADISLR
VGGRRTWRTLPFILLFPVVIDLVAYGTAWITGLVQFVTPDTYIKAPPVVIFLGLLLTQMVIGTITGLIGSVGEELGWRGY
MLPRLIDARVPYPMLTSGIIWGLWHLPLMIAGNYYSGPYLALSAFLFMISVTSFGYIIGCLRLTTGSVWPAFFLHAYWNA
VIQDVFDSFSKGENVLLWTGESGILVALALLAAAWFISRRPMSMRRL
>lcl|NZ_CP026520.1_prot_WP_042233579.1_5115 [locus_tag=PC41400_RS25740] [protein=DUF3243 domain-containing protein] [protein_id=WP_042233579.1] [location=5670452..5670805] [gbkey=CDS]
MTEHNHMVDKNGTLDTGKVNDVMNRMDTGEKEQILKNFEEFKSYLGSRVHLGESLGLGEEQLARTAEKVAGYLAAHEEPR
NAEENLLRELWNVGRDEERHMLAHMLVRLAQKANKAN
>lcl|NZ_CP026520.1_prot_WP_042233582.1_5116 [locus_tag=PC41400_RS25745] [protein=YutD family protein] [protein_id=WP_042233582.1] [location=complement(5670916..5671647)] [gbkey=CDS]
MIHIGGRAYELVTEHRNGWNQEVFRDRYSEVLERYDYIVGDWGYSQLRLRGFFRENHSKAGKESSIVYLQDYLNEYCNFG
CAYFILEKLPGKHTPPEQTEEEAAAAEGTPKPSRHQGQGQGASRAEAKQESREGKSETREPRADHKQEARESRQEQRGSS
NRSQRQQDGGRGDAQVSQKQGKPRPPRQPRGDQQRGERSTDGKPHNQRQGGGQGQNNNQNNNRPKPSQAAETPAPPTGPE
SSE
>lcl|NZ_CP026520.1_prot_WP_042233584.1_5117 [gene=lipA] [locus_tag=PC41400_RS25750] [protein=lipoyl synthase] [protein_id=WP_042233584.1] [location=complement(5671680..5672564)] [gbkey=CDS]
MATKPLQRKPEWLKIKLNTDDNFKDLKNMMRSKTLHTVCEEAKCPNIHECWANRTATFMILGDICTRACRFCAVKTGLPT
ELDLQEPERVADAAAQMGLKHCVVTSVARDDLKDGGATIFAETIRAVRSKVPFCGVEVLIPDFMGDEASLQIVMDAKPDI
LNHNIETVERMSDHVRAKAKYSRSLELLARAKKMNLDIPTKSSIMIGVGEEWDEILQTMDDLRRHDCDIMTIGQYLQPTR
KHLAVEKYYTPEQFALLKEEGMKRGFKHVESGPLVRSSYHAHEQVRSASQTSAP
>lcl|NZ_CP026520.1_prot_WP_042233586.1_5118 [locus_tag=PC41400_RS25755] [protein=M23 family metallopeptidase] [protein_id=WP_042233586.1] [location=5672822..5673859] [gbkey=CDS]
MKGILPSKRLLAGLLTMCLAVGPLTATAAEPVKTPPPAAKPQQPPDAERKELLQKVSEITGIPWPLLAAVNRYEYTMNKI
RKKDAAERLIDIDFPELTWVGMLNPDHEDTDPGSISFFGGVGVDGSGDGLADRGNDLDRLASMARLMLRHGNDPEDIRIS
LWEHYSNTRSVERIEQFADIFEKFGTLSLHEHAFPLPVKAEYSYRSTWGASRGWGGFRIHEGTDLFAGYGVPVRSTSYGI
IEVKGWNPYGGWRIGIRDLDNIYHYYAHLSGFDKKIKEGDVVAPGQVVGWVGSSGYGKPGTSGKFPPHLHFGMYRDRGLA
DWPFDPYPSLKKWEREEFKKRAAGK
>lcl|NZ_CP026520.1_prot_WP_042233588.1_5119 [gene=yunB] [locus_tag=PC41400_RS25760] [protein=sporulation protein YunB] [protein_id=WP_042233588.1] [location=complement(5674009..5674821)] [gbkey=CDS]
MIRQRWKSRPTRKQGWKKRLFIVLLIFMLVSLQSFIMIERNLKPPLMNLAKVRVKQIATESINEAISDRISRGTDFEKLI
DWKTDSSGKVSGFMLNYSEHMRITAETRGAVQSQLNRLTAIPEHIPLGQAMGSAILASFGPNIPIRITPAGAVKVDLSTR
HENAGINMILVEVYIRIVAEVSVIIPFDSEPEIVETEIPISYSLVVGDVPMYYFDNKGNPVGSSKDAVPPSISLPNLKGQ
PTAGSGEKQQPGAHTEGVTKPEGAGEKKAE
>lcl|NZ_CP026520.1_prot_WP_042233590.1_5120 [locus_tag=PC41400_RS25765] [protein=hypothetical protein] [protein_id=WP_042233590.1] [location=5674967..5675455] [gbkey=CDS]
MLTFEQKIAIMESFPELQRKNVSLGRVNFHYEDSAYDKKTVGYHFHPNGNGYIYAGQIEGAETDDKGLVNIRDYSEEQLR
DLAGAAIRSLTAGAGTSVTADDKTGSSFPDEQTWVNADGNVLTVKLDDEMWFIHAGLNLDSVFESYEEAVEYLEEEGFAR
KK
>lcl|NZ_CP026520.1_prot_WP_042233592.1_5121 [locus_tag=PC41400_RS25770] [protein=RNA methyltransferase] [protein_id=WP_042233592.1] [location=complement(5675645..5676466)] [gbkey=CDS]
MAQAYNQPDITSVQNNRVKEWAGLLDRKGRQKAGAFLIEGIHLVDEAVRAGAPVRTLVYEADKGLPGTLADHAEAAGIET
VSVTRPIMERCTDTETPQGIFAVVDKPAEPAETLLADTGSDLVVVVDGVQDPGNLGTIIRSADAVGASAVVLGRGTVDLY
NPKAIRSTMGSLFHLPIVEADLQELLPAAREAGRQIVVTSLQADSSCYALDMTRPTWLVVGNEGRGVSDAAGELATHRVI
IPMQGQAESLNVAMATTVLLFEASRQRLAGSSV
>lcl|NZ_CP026520.1_prot_WP_042233594.1_5122 [locus_tag=PC41400_RS25775] [protein=TrkA family potassium uptake protein] [protein_id=WP_042233594.1] [location=complement(5676474..5677133)] [gbkey=CDS]
MKKQFAVIGMGRFGSSVAKNLTGLGFEVLALDQSEERIQEISTVVTHSVQVDCTDEEALRALGIRNFDVVVVAIGENIQA
SILATLLLKDIGIPTIVVKAANDLHGKVLRKIGADKIIYPERDMGQRVAHHLISPNILDYIELSSEHSILEMKANDRMVG
KNLRDLDLRAKFQCNVIAVKRQEQLIIPPPVETPIQPGEILIVLGNNKDLKRFEVEYSE
>lcl|NZ_CP026520.1_prot_WP_042233596.1_5123 [gene=sspI] [locus_tag=PC41400_RS25780] [protein=small acid-soluble spore protein SspI] [protein_id=WP_042233596.1] [location=5677248..5677451] [gbkey=CDS]
MILSLRQAIVQRVEGKSNDELKEIIDDSIDGQEQALPGLGVLFEVIWKNIDSEKQSELVTALHDHLT
>lcl|NZ_CP026520.1_prot_WP_042233598.1_5124 [locus_tag=PC41400_RS25785] [protein=MtnX-like HAD-IB family phosphatase] [protein_id=WP_042233598.1] [location=complement(5677564..5678286)] [gbkey=CDS]
MEKGQLVVVTDFDGTLMEQDVGATLMKELGVEDRPDVRGELERFLNKEIGSREWIHTSYGLLGDRQKEVNAIADRMNLRD
GAEEFLAYCKNKSIPVTILSDGMKYYIDRIMAANHVTVDSVISNPIQYGEDGAYTLGTQNTNPACSWCGCCKSGVVRKLK
EEGNRVIYIGDGSSDIYGSAYADWVFARSSLARYLTREEISFYPFATFHDILKVVREQEENFLTGSPERREPAGHSFCKF
>lcl|NZ_CP026520.1_prot_WP_042233600.1_5125 [locus_tag=PC41400_RS25790] [protein=LacI family DNA-binding transcriptional regulator] [protein_id=WP_042233600.1] [location=complement(5678508..5679551)] [gbkey=CDS]
MVTRKDVAKMAGVSEATVSRVFNNVGPMKESTKQKVLAAAKELNYHPNAVARSFALGKSGHIGLILPYVPKVHLFSTYYF
SEILSGIGEGVRRHGCDMLVMFRSPHESVDYTEPYRSRKVDGCIFLGALDVPAQREGFRELDRQGMPFVLINQHYEGETY
SVVDADHEQGCYEAVKHLLEGGYRRIAFLNGSPQYSNSADRLRGYYRALNEAGLAPEPELILEGNYSRTSGYAAAARIAE
LGGRVDAVFSSNDRMAVGLMQGLKERGIVPGEDLAIVGYDDSEAAKLSDPPLTSVYVPFFEMGLLAADALLKRINASPEE
DEPPVSFHERLSTKLVVRASSNPRPGA
>lcl|NZ_CP026520.1_prot_WP_042233603.1_5126 [locus_tag=PC41400_RS25795] [protein=response regulator transcription factor] [protein_id=WP_042233603.1] [location=5679912..5680637] [gbkey=CDS]
MKLLVVEDDRSVCEMLRPFLEREGTVTFVHTGSEAESLIREASWNLIILDWMLPNRSGIELCRMIRKTQDTPVILLTARG
EEGDRVLGLESGADDYITKPFSPLELLARIKAIMRRYHSGPGSMADNGAAESAFPEELVEDNGERLVHKDIKVDTSTREV
YVGPDKFTNLTPREFDLISLLVRYPKRVFTREQLLEAVWGYDFYGDERTVDVHIKRLRTKIGSAERPLIATVWGVGYKLE
E
>lcl|NZ_CP026520.1_prot_WP_042233605.1_5127 [locus_tag=PC41400_RS25800] [protein=HAMP domain-containing sensor histidine kinase] [protein_id=WP_042233605.1] [location=5680639..5682135] [gbkey=CDS]
MKFRLIPTRHRFTFFQRQFVSLLFLSLLVLVLVSFIFVYLFKIHIYDEKTEELESAGDTISRLILREEVVPELTLSAYRT
VLRERGISFIAMNAKGQLYFKDQRTTPPTFRSKTFMDSLLAQIPKLQSGKALIVEESGPDPYLVYPYELKLKHSDGKNYL
FVMSPVKGMTQTLSRVYEMVVLAGIVVFVLAMIVALIVSRNMSQAVLSIRQATRRIADGRYDTRSDLARTDELGDLSRDF
NRMAEQLEFTSHKLERIEIKRRRFITDVTHELRTPLTSIRGIVEGLKSEYITGHEEQVKYYGIIEQETLRLIRLINELLD
MEKIQNGLITLRREYYPLNDLLEVLKESFEFLIGDKDLLLEIDCPPDTMIYGDYDRLTQILINLIKNSIQFTSHGTIRMT
ARQTETATELSISDTGRGMTAEEMEQIWDRFYKADPSRSKDRSETGLGLSIVKRLMEAHDAEIRVSSVPGQGTIFRLRFP
VKPGKAAHTGELPGEGEV
>lcl|NZ_CP026520.1_prot_WP_042233607.1_5128 [locus_tag=PC41400_RS25805] [protein=YheC/YheD family protein] [protein_id=WP_042233607.1] [location=complement(5682249..5682998)] [gbkey=CDS]
MRRYVLSKWVKTEAVLHSTAVRPHIPPTRILSEESLRDMLDKYRMVYVKPDCGTFGRGVMRVDYNEQDAKPYRYRHLKTT
RSFQTYEDLYQSLKTATDGRKYLAQKGIRMLKHDKNVFDLRVIVQTKPGGQWEVSGMIGRVSQTGKVVTNYHSGGKLRSV
ETLLSPYFGAKELKRFKARLKRLAHSTALALHSKYPGIREIGLDVAVDQELKPWVLEVNTKPDPYIFKVLKDKRIYAKVL
RYYKYNKRK
>lcl|NZ_CP026520.1_prot_WP_042233609.1_5129 [locus_tag=PC41400_RS25810] [protein=hypothetical protein] [protein_id=WP_042233609.1] [location=5683173..5683877] [gbkey=CDS]
MRRKYVSVLMILIGLMINPYHANALDYAFIPLKELAEDSDVIVTGTVLREEKSDSDNHYTLQPDQILKGGMDVEELIRIK
VLQFADEGKMFPGERYLLLLKAGNPFIVNGVHQGLIQLSGGKPFSRFYKPEEVTALLDKLGAKIKKSSSAVTSSPSPGNV
SPLPESVSVGVRPAETDSGPWEERKPADPFVLSEWYGPLFLAAGLAAGIITAFAAEKRARKQEAETENQDPAKH
>lcl|NZ_CP026520.1_prot_WP_042233611.1_5130 [locus_tag=PC41400_RS25815] [protein=peptide chain release factor 3] [protein_id=WP_042233611.1] [location=complement(5683967..5685556)] [gbkey=CDS]
MSTPLQKELQTEAAKRRTFAIISHPDAGKTTLTEKLLLFGGAIRLAGTVKGRKVGKYATSDWMEIEKQRGISVTSSVMQF
DYLGHKVNILDTPGHQDFSEDTYRTLTAADSAVMLIDVAKGVEAQTIKLFQVCRMRGIPIFTFINKLDREGRDPFELLEE
LEQVLGIRSYPMNWPIGMGKRFCGVYDRQRTQLELFQGDDHKQVAVRKVEGYEDQVIKEIAGEHLYEQLGQEMELLDVAG
DPFDYEKVMRGELTPVFFGSAVNNFGVQTFLENFLELAPAPEPRKNSQGEMIEPTKENFSGYVFKIQANMNPAHRDRIAF
LRIVSGKFQRGMSVKHVRAGKEIKLSQPQQFLAQDRDIVEEAYPGDIIGLFDPGIFRIGDSLSQKEEIVFDELPTFSPEI
FAKVTVKNALKHKQYQKGIDQLTEEGTIQVFRTTGAFDDTILGVVGQLQFEVFEHRMKAEYGVDIQLQRLGFQFARWIID
EKIDPSRFRINSQLVTDKKGNYVALFESEYALRSAMEKNPTAQFLTNAP
>lcl|NZ_CP026520.1_prot_WP_042233613.1_5131 [locus_tag=PC41400_RS25820] [protein=metallophosphoesterase] [protein_id=WP_042233613.1] [location=complement(5685624..5686706)] [gbkey=CDS]
MDRRSFLKWLTAFFAILLGAGAGTVLSGLRRKSGLSDRESAVVQAEEAEELPAGEDLQLPDDPQAGKALATFAIMSDIHA
SPDDPLTLEHFKSALSDMTAMKQPVDVLLLTGDCTDYGRERDYKELGRLMKAYKLPPVHANMGNHEYYDLWLDKTGGFNR
EGMPNGKTDAMARERFLTFFGYKQPYNRMDLEGCTILLLSQEGYMQEKPEAGEGAWYTDAQMEWFKTQLAGLDKTARIFI
MIHQPLPMQGRSGGSHQIVRGKEIRELIAPYKNVFVFCGHRHQDFQNGSAHYLQEPFHYFHNSSVGRTLNKALPDARKSK
TQGLYVQVFENKVVVRGREYAQRRWLKEAHWTIALNKKSV
>lcl|NZ_CP026520.1_prot_WP_042233617.1_5132 [locus_tag=PC41400_RS25825] [protein=methyl-accepting chemotaxis protein] [protein_id=WP_042233617.1] [location=5686899..5688365] [gbkey=CDS]
MESNHLLFKRNKLLNRILWIMLALGIVIDLMTASLEVVLVLAVVGIVICGISTLMSVKHIAPQLVMYIVPVNVTVLTTLL
IYAEPLTSTYYLIFVNIALMTLYSNYRPIVFSGILGGALTAFLFIQPRFYEAMFPKESLANLLLYVAMITAASAASARFS
EKLQTEVLEKQAETLDAKAQTDSLLEQVGSTVQVLNEFNDTLKTNVATAGKISQEVTATFGEVSTSVEHQTQIVVDISGF
VQEVERVVSSVADGSGTLQRLSTENIGLTSQGSEQVHALSAEIETVYSIIHRTVHLMNDLNEQGVRIGSMVGLINDITSQ
INLLALNAAIEAARAGEHGKGFAVVSDEVRKLAENSRRSTEEINDILQGIRHTSIEVTKQVNLGRQAITAGRAANEQVES
IIQRVADNTGLVKNQSDHVEESVALLQEEYRKISHEIVAIAGSTEQNMASFEEIMAGLENHDARIKDIVTSYQRLDSLIT
NLKAMTRG
>lcl|NZ_CP026520.1_prot_WP_042233620.1_5133 [locus_tag=PC41400_RS25830] [protein=GNAT family N-acetyltransferase] [protein_id=WP_042233620.1] [location=complement(5688411..5688974)] [gbkey=CDS]
MYRKGDSLDDFILFDRIWTRVWQQQGFELEFTRRDADCYLILSDDGEVAGTVEFKRYDPGRHPIDALAPFSEHPALAPGN
AVEVDKVALLPEYRGRNLDKLISLIVHYCENRGRTHIVALMEPKLCKALRMAFRIKLERIAEPFHYKGGMVVPVVCDVSR
LYLNKPAYGWLLPNPEQPEIILPAVGS
>lcl|NZ_CP026520.1_prot_WP_042233622.1_5134 [locus_tag=PC41400_RS25835] [protein=ABC transporter permease] [protein_id=WP_042233622.1] [location=complement(5689148..5691085)] [gbkey=CDS]
MNFRQLALKNIRGNWHQYGAFFWSSVFSVTIFYIYSAFILHPDVINGSMPAASKIREVMIACDYIIVIFSFFFILYSMSA
FFKSRKKEFGLLTLFGMTKGQIRWMVLLENVIISILAVACGIGIGSLFSKLFFMGLARLLEVDSPIRFVIPVKAVLLTVI
GFLLLFLTITLLTLYQVGRSEIIDLLRASKQPKKMPAFSIWLVVLSAVCLVAGYTLAYQITLMNFILYFMPVVFLTVLGT
YFLFTQGSVFILRRLQRRTSLYYNQTNLITISQLTFKLKDNARMLFMVSILSAVILSASGTVYVILKDTRNQLMEHYPQS
IAFAEKGLDSHRVLDPDVLKSTLKQNGVELDYEVRMAGIPVRIPVELGGRVREQQLFIVSEEDYNTQAKRQGKPELRVER
GHASYIYPYKEQQRKSFTPGQNLEAKLKTGTRTFTLDEQINGSVVPTQFDISTLLVTDASEFKEMMDATPNEDKSVLYGY
EIKNWEEASPTVHQIEKMVPEQEKQNFGTRVNQYLETKQLMSLMLFIGIFVSFLFFIASGSMIYFKLFTEIEEDQAQFRA
LTRIGITDKEIRRTVSKQVGMIFFIPLVVGLVHTSFAMKALGNMLISNIWPYAAVVFVIFALMQTLYFLAARRSYMKRIL
AHSVR
>lcl|NZ_CP026520.1_prot_WP_009673076.1_5135 [locus_tag=PC41400_RS25840] [protein=ABC transporter ATP-binding protein] [protein_id=WP_009673076.1] [location=complement(5691060..5691836)] [gbkey=CDS]
MAVLKAEQVTKIYGTKSNLTYKALENIGLEIGEGEFVGVMGPSGSGKTTLLNILATIDKPTSGHIEINGTDPAALSGKQL
ALFRRRELGFIFQDFNLLDTLSIRENIILPLALDKKPVAEIESRLKPCAELLGITAILDKRTYEVSGGQKQRAAIARAII
HRPSILLADELTGNLDSKAAKDVMESLKELNEQLGATIMMVTHDPFAASYCKRIVFIKDGKLFSELRRGANRQAFFQQIL
DSLSVLGGNFDEFSTARP
>lcl|NZ_CP026520.1_prot_WP_042233624.1_5136 [locus_tag=PC41400_RS25845] [protein=sensor histidine kinase] [protein_id=WP_042233624.1] [location=complement(5691931..5693070)] [gbkey=CDS]
MHDKAWTRKRFLFDRLLYTAAFFFAILLVLVVLWLDRMEAGLPVKPDNLVYLFLLAGAVWIGVLIYDYVRQAGYASALNK
RLGESAQSSLDSALFLPSRTMEQQAVNELLQKQHRAYLSTLSSYRASQEQHLHFTNQWVHQLKTPVSVIDLLIQQSEGTL
TPPERIQELLTSIQEENEKIASGLQMMLYTARLEKFELDVHPRRIELTAVARRIVNDYKKACIRASIFPKIESSAPDIYV
ETDEKWLSFVLHQLIGNAIKYSRKREGNKPLLIRIEQKSYGTRLEVQDSGIGIAAEDLPRVFDAFFTGENGRGVVTESTG
MGLYLVKQVMSRLGHQVDIQSEPGQGTTVSLMFQSQGLHRFGERDEAGGVRPGSRVTKM
>lcl|NZ_CP026520.1_prot_WP_042233628.1_5137 [locus_tag=PC41400_RS25850] [protein=response regulator transcription factor] [protein_id=WP_042233628.1] [location=complement(5693063..5693773)] [gbkey=CDS]
MYRILIVEDDDKIAHILKDTLTRYEYEVFRAVRFHDIKGEAAELEPDLILLDINLPRYDGFYWCRQIRTFSNVPIIFLSA
RTGEMDQVMAIENGGDDYLTKPFNLDVVLAKVKGALRRAYGEYAQAQPSRDEWELNGLKLDRMKNTLEFAGKRMELTKNE
LLLFDCLAHRAGSIAGREELLEALWDDVHFVDDNTLTVNVTRVRRKLEEIGLPGIIETVRGQGYRLQTEWGNTGDA
>lcl|NZ_CP026520.1_prot_WP_042233680.1_5138 [gene=zwf] [locus_tag=PC41400_RS25855] [protein=glucose-6-phosphate dehydrogenase] [protein_id=WP_042233680.1] [location=complement(5693912..5695429)] [gbkey=CDS]
MNGAVFLIFGATGDLAKRKLFPAFYSLYREGKLNEHFAVIGLARRSRTNEQFREDLRDSINEFARHHVQDDEDWAKFAEH
FEYLPLDINNVDGFRQLNTLTQQLDVKFQTNGNRLFYLALAPELFGNVSYNLREGGLLETTGWHRLVIEKPFGYDLESAG
KLNQQLRQVFEEKDIYRIDHYLGKEMVQNIEVVRFANAFFEPLWNNKYIANIQITLGETVGVEDRGGYYDHSGALRDMAQ
NHMLQMLAMMAMEPPSRLHPEDIRDEKVKVLRSLRPFESAEEVTKNVVRGQYTAGQLNGKDMPAYRSEESVNPESNTETY
LAARVFVDNFRWAGVPFYIRTGKRLPVKSTEVVVEFKNVPDNVYLANKHDLQPNLLVFRVNPMEGIYLQMNAKKPGAEGV
IIPIAMDFCQSCQVGLNTPEAYERLIFDAMRGDSTYFTRWDEVAHAWQFVDAIAQAWKENSGDLAHYPAGSWGPDAANEL
LAADGFKWWPVNGQHEGEVIWASQS
>lcl|NZ_CP026520.1_prot_WP_042233630.1_5139 [locus_tag=PC41400_RS25860] [protein=hypothetical protein] [protein_id=WP_042233630.1] [location=5695589..5695909] [gbkey=CDS]
MQRIDPASREQIKPHIGRPILVYLRDGTEIFGILSRLDRGNLILNEKPSVKNLSPKTTKKIKVAKNGGKKSVKLAAEESQ
PVIAHPEGYYPLVLDRNFIAALIVLQ
>lcl|NZ_CP026520.1_prot_WP_042233632.1_5140 [locus_tag=PC41400_RS25865] [protein=GNAT family N-acetyltransferase] [protein_id=WP_042233632.1] [location=complement(5695997..5696434)] [gbkey=CDS]
MIRKRIPAKDDRILVRLTEKELLPYTRMTVPEANVNLSEMKKRFKLGETFVVDEGSSAAGFVSVQVSDDSLFVDMLAVDS
RFQGRGLGTSLMEHAERYGRKLKCRASRLYVDSVNEKAISFYLSKGYTIKSYVPQLRIHLMEKGL
>lcl|NZ_CP026520.1_prot_WP_042233634.1_5141 [locus_tag=PC41400_RS25870] [protein=hypothetical protein] [protein_id=WP_042233634.1] [location=5696890..5697348] [gbkey=CDS]
MYIVTPQYVNETYRSHSNLFRIQAIRRMNGLGPVSPRGSFPEMLYTRQDRSGKAGQAEQGQTSGFLAEQRDAASGKEARQ
QSGMPAMLRRLGETNEWKRPQDDDRPGPAEQSEEAEESLVQANAAMQQMKLNKALQAYRPFIRKQFALESAV
>lcl|NZ_CP026520.1_prot_WP_036739861.1_5142 [locus_tag=PC41400_RS25875] [protein=response regulator transcription factor] [protein_id=WP_036739861.1] [location=complement(5697398..5698033)] [gbkey=CDS]
MVKVLLVDDHEMVRIGLAAVLGTEEGIEVVGEASNGEDGIRLAQEYKPDVVLMDLVMEGMDGIETTRKLLQLYPDCKVIV
LTSFLDDEKMYPVLEAGAFSYLLKTSRATEIAQAIRAAAKGQSILESQVASKLMNRFRQPKPAAAPHEDLTEREMEVLRL
IAAGKSNQEVADDLFIGIKTVKFHVTNVLAKLGVEDRTQAAIYAFKHGLAE
>lcl|NZ_CP026520.1_prot_WP_042233635.1_5143 [locus_tag=PC41400_RS25880] [protein=sensor histidine kinase] [protein_id=WP_042233635.1] [location=complement(5698039..5699097)] [gbkey=CDS]
MDSYKKRLTNMIWRSMAESIVLCLAVLTFIVYVLYTAKFLTPPVPWTQGIKIGAMLVGGSIILGSAYGYWSSNRTTRRLE
LLMETMQMLEIGSISRSTLPPLGEDEIGRLGEQLGRILKKWEEQVASLQRLSNNNAQLAEKARVSAIIEERQRLARELHD
AVSQQLFAISMTATAALRTLDKDFEKAQRQIHLIEEMSSVAQSEMRALLLHLRPVHLEGKQLSEGLIRLLQELKAKVPMD
IIWEVDDEIALSKGIEDHLFRIVQEAMSNALRHSKATRLELRMQQKSGWVRLLIRDNGVGFDLDEEKQTSYGLSTMRERV
NEIGGSLQLVTAPGSGTRIEIRVPIMTEGEEA
>lcl|NZ_CP026520.1_prot_WP_042233637.1_5144 [gene=liaF] [locus_tag=PC41400_RS25885] [protein=cell wall-active antibiotics response protein LiaF] [protein_id=WP_042233637.1] [location=complement(5699139..5699801)] [gbkey=CDS]
MEQQMKNNRHRNTAMLLIALGVFLILERWVGIFVVIALLLMFFGFYKVRYENEKKGVAYMGVGAVILLGNNFSLVLAVIL
ISLGIFYIKSKKVQPGDDFIQKQTLIDSIRPGKEPWVLKDSSIWFIIGEAHLDLSYAILEKKETTLVLQGIIGDVRLIVP
EEIGVSIEASVTVGQISSPLEKESGIINKLNWISPNYETREHQVKLIVSYIVGDIDVRML
>lcl|NZ_CP026520.1_prot_WP_042233639.1_5145 [locus_tag=PC41400_RS25890] [protein=hypothetical protein] [protein_id=WP_042233639.1] [location=5700116..5700733] [gbkey=CDS]
MTKLNFRARTALIALLCTAAAGFVPASAVSAQAAGTGLSSFDTSGAGSSGIILPGGPWTARLQATPSHAVPPSSGAFSRE
IRAQLYDVDKQKVVETLIVTDEIRTEASRWLASIKDIAVQTKIDNVQGVVLKLPLDPPLAINNRWMQGSSNELFLFLDPS
RLNEPLLLVFSTTGKPFLFKTSVQVKPFLEKERLLRYLKPVPASG
>lcl|NZ_CP026520.1_prot_WP_042233641.1_5146 [locus_tag=PC41400_RS25895] [protein=hypothetical protein] [protein_id=WP_042233641.1] [location=5700865..5701803] [gbkey=CDS]
MREDWKYRYDEVLKKESLLQTVNKQLEKAQVRLKQAGEQRTAAEKALKSETRDVERLEKASLTNLFYTVLGRKIEKLEKE
EAEAIQARIRAEETRLAEEELARMTAELEQRRSSLLGWEEEAASLLQEKEEWLRGRSEDFRRQLNRLYEAKRQALSEIKE
LDEALRAGYELVESLEEAENLLRSAKSWGTYDMFGGGVIATHIKRSKANEAMEQIEYAQGLMSRFKRELEDVGGTAEMNL
DYNGFLGFADYFMDGFIFDWIVQGRISDLKDAVSENIGRLRQLLHALRQKYEARLTDEAGLTRDIHLLIERS
>lcl|NZ_CP026520.1_prot_WP_042233643.1_5147 [locus_tag=PC41400_RS25900] [protein=GNAT family N-acetyltransferase] [protein_id=WP_042233643.1] [location=complement(5702040..5703239)] [gbkey=CDS]
MSEIRRLRREELRDSLDLSMYAFQYELPEEELVERLATMNPEETWGYFIDGKMASKLTILDLTTWVNGKKLKMGGIAGVA
SWPEYRRQGMVGQLLSKALEVMKENGQTLSYLHPFSFAFYRKYGWELAMTFKKIEIQTSQLPKLGAAEGRIIRTAPSAAA
LNPIYDAYAARFNGPITRSEEWWETRLLKKKGHMAVYLNAAGEPKGYVHYWIENQECKIHQLVYLDQDAKRGLWKFIGDH
DSMFDKLIMQAPPDEDLAFEMDDPRIKQELVSFFSARIVDVEGLLGQLQLRHTGLPQEASLRLHVTDGQAPWNHGTFELR
LNAPEGSAVVVKTADEPAGGSGAAEDGLACDIRTLSVLLLGAKSAGELHRIGRLQGDERSVELLAVVLPRRTCYLPDFF
>lcl|NZ_CP026520.1_prot_WP_042233645.1_5148 [gene=moaA] [locus_tag=PC41400_RS25905] [protein=GTP 3',8-cyclase MoaA] [protein_id=WP_042233645.1] [location=complement(5703285..5704295)] [gbkey=CDS]
MTSIGLLKDKFNRIHDYLRISVTDRCNLRCVYCMPEEGMTFEPDERLLNYDEITEVVRVLAEMGVRKLRITGGEPLVRRD
LYVLIRQLAAVPGIEDIALTTNAVFFAQQAERLREAGLTRINISLDSLRPDRFKLITRGGDVARVMAGIEAAQRVGLYPV
KLNVVLMQGLNDDEIDDFLQMTVDKDIQVRFIEYMPIGHQDDTWKLRYVPLSSVLDRCSERGWKVEEVQSVKGNGPSQNY
RIAGGRGSFGLIHPVSDHFCASCNRLRLTADGNIKPCLYWSHEFNVRPLIGDDRAIRELFYQALDVKPESHEMAKALQGD
EPSHLPTVRRMSQIGG
>lcl|NZ_CP026520.1_prot_WP_042233647.1_5149 [locus_tag=PC41400_RS25910] [protein=MOSC domain-containing protein] [protein_id=WP_042233647.1] [location=complement(5704292..5704978)] [gbkey=CDS]
MTPQIVSINVGKPKKIVYQGKEVETGIYKEPVHRPLFLSKTQLEGDGQADLEVHGGPDKALCVYSANHYDYWEKELGQPL
SFGAFGENITISGLEEEEVCIGDIYELGEALVQVSQPRRPCFKLSIRYSDVKLPVKLQDTGYTGFYFRVLKEGIIPENPV
FRLVTKHAQGLTVEFVNTCKYHDKNNIPGIRIILANEALAESWRGSFEKKLAEAVDLSDPAQSGGVKE
>lcl|NZ_CP026520.1_prot_WP_016423567.1_5150 [locus_tag=PC41400_RS25915] [protein=hypothetical protein] [protein_id=WP_016423567.1] [location=complement(5705095..5705466)] [gbkey=CDS]
MAKKSWSTQEMERLCMLSKECPPSVIAKELARPVTSVRKKLREIGVTYVKGDEWKRKQVQMLISDEARGELPRRNKQELT
EEAQVLNEFWTTLLSMASVAKKKGKRVDVLSFIDTYRKIKLGG
>lcl|NZ_CP026520.1_prot_WP_042233649.1_5151 [locus_tag=PC41400_RS25920] [protein=fumarylacetoacetate hydrolase family protein] [protein_id=WP_042233649.1] [location=complement(5705770..5706720)] [gbkey=CDS]
MKLLTFLGEQGPRLGIRLEQGVLDVAEALKNMKGYEEGSVPTDMDQVIKGGPEAIQSLQKVIDDLPDTSENEPYLLEESS
LVFAPCVTEPGKIICVGLNYQKHAEETKSPIPEHPVLFNKFNNALTGHRCDVSLPNVTSELDYEAELVIVIGQKAKDVSK
EEAAQYIFGYTCGNDLSARDLQRRTNQWLLGKSCDGFLPLGPYIVTGDEIGNPNELKIRSLVNGEQRQNSTTADMIFKVE
ELVSYISRHMTLLPGDIILTGTPEGVTIGYPREQRVYLKDGDQLSVEIEKIGTLSNKIVAETKVEEGKPVGRTLGV
>lcl|NZ_CP026520.1_prot_WP_042233651.1_5152 [locus_tag=PC41400_RS25925] [protein=alpha/beta hydrolase] [protein_id=WP_042233651.1] [location=5707793..5708578] [gbkey=CDS]
MFTTIHGLKTFYEERGSGPAVLLLHGFPLDHRMWQEQISALESNYRVIAPDLRGFGQSEPHKDLITLEHMADELLLLLDQ
LQVEKAVLAGFSMGGYVAFSLLRKAPNRFNGLILANTRPDADPLEGQINRMKMAASLLEQGPQAASKAMIPKLLSEGEGS
GLASEIQDQINGMSPLGLVHASLAMAFRKDSTEPLSTIQAPTLVIGGEKDSITPPSIMKKMADSIPNATYSEIKDSAHLT
VREHPDQVNKAITDFLSHLHS
>lcl|NZ_CP026520.1_prot_WP_042234475.1_5153 [locus_tag=PC41400_RS26025] [protein=hypothetical protein] [protein_id=WP_042234475.1] [location=complement(5715604..5716191)] [gbkey=CDS]
MDRQLYLEDSRERTPSLPEELRDRQLLSEFSLFLNKYLKSKSYILEEHLLDAYQNVLEALQHWARIVIIEEGETVQEAVW
NQVRSINSGVYKLYEELTTSKETLKQRIQLVLLACEFSVMSKMERCCKPLIQRLDSRPEPWSADELLEQPEIQILGNNLH
QLLNKLVKKTLVKEVAVPADAECSQLLLRYTLFKN
>lcl|NZ_CP026520.1_prot_WP_042234477.1_5154 [locus_tag=PC41400_RS26030] [protein=DUF2614 family zinc ribbon-containing protein] [protein_id=WP_042234477.1] [location=5716607..5717020] [gbkey=CDS]
MLLKSSKINEFRLWGLLLTMGGMLIMIIGLAGVVFDWGAVGRIIAVIFLILGMISMLGSMAIYFWAGMMSTSAVALDCPE
CGKRTKMIGKTDRCMFCKTILTVDPSMAPAEAEISDEGNIRQPETGSATTAIDSSKN
>lcl|NZ_CP026520.1_prot_WP_241688739.1_5155 [locus_tag=PC41400_RS26035] [protein=glycosyl hydrolase family 18 protein] [protein_id=WP_241688739.1] [location=complement(5717161..5718867)] [gbkey=CDS]
MKQPYMHGSKRKKSRFRSFILLLIVLAAIAYLVMNRLPNNNHITPFYAKLDKPVFYQGEAYESSALGSQEAMKLPLGLIT
KWIDPAVRSEASGNSVILTTEQAVLRLQADKVSASRNLKEIPLVSPMTKQGKEFYVPVSVLKELYGIQVREDAVTGAVSL
FNPGEKIRWGTFRGKEGGELRSGSSIRDPITGSLVDGERVMIWGEQNGWYKVQQTGGRAGFVKASHVTGEKGEEIPEFQP
PAAKPRWKPQGKLNMTWEHVVTATPKTASIPEMPGLNVVSPTWFHLADEKGTIKNQADSDYIKWAHGRNYQIWALFDNSF
EPKMTGKALESYETRLNMINQLLSFTKQYNLQGINIDFENMNKSDGPNLVQFVREMAPLLHERGLILSIDVTAKSSSDAW
SRVYDRPALSAAVDYMMLMAYDEHWASSPKSGSVASLPWVERSLKQILDEDKVPPSKLVLGIPFYTRIWTEEQKDDGSTK
VSSKAVFMEKPRKIIQEKGLTPEFHEDIGQNYVEYKEDEKLNRIWLEDETSVKARLEIVNKYKLAGVASWRRGFEEPAIW
KVMADALQ
>lcl|NZ_CP026520.1_prot_WP_042234478.1_5156 [locus_tag=PC41400_RS26040] [protein=Fur family transcriptional regulator] [protein_id=WP_042234478.1] [location=complement(5718978..5719412)] [gbkey=CDS]
MSEHLEEALAKLKTTGVRMTPQRYAILTYLFDSFNHPTADEIYKALESKFPNMSVATVYNNLKVFIDAGLVRELTYGDDS
SRFDADLSDHYHARCQVCGKISDFDYSPIVDVEHVAAQQTGFRVGGHRLEVVGLCPDCVGKATH
>lcl|NZ_CP026520.1_prot_WP_042234480.1_5157 [locus_tag=PC41400_RS26045] [protein=DUF4097 family beta strand repeat-containing protein] [protein_id=WP_042234480.1] [location=complement(5719534..5720763)] [gbkey=CDS]
MIGLIKAGRFTAAVLFVTVGVSMLLDRAAGTHMVGQLIRWWPFLFVAAGLELIVLQFLSKSRETAVKLDPVAIFFSVIIC
AVVVGSQGFSQGLLSGLNFNVDAVQGRSFDKEPTELTLTPAVTRIHAINTNGRIELAAEDGDAVRIQATVWVNDDDEQEA
RKIADASHVEQSIQGAELTLKAVGEEYTKAWLVKNKARIDITVTLPRTASLDYLAETENGSIRAAGLRTPNLFSAHTSNG
SMELTDIQGNVEAVTENGKISAGSISGAVKAQSVNGEIVVRDTGAGVNVAASNGKVSLSSVKGDAVARTDNGSITIKNVF
GRVEAHTENGAVRLESAEMAGDWQASTDNGRITIRLPGGAGFKVQGSARVLHTGLPIRLEEEKLSGTVGSGEHILAIRTD
GEISLETVN
>lcl|NZ_CP026520.1_prot_WP_053228785.1_5158 [locus_tag=PC41400_RS26050] [protein=hypothetical protein] [protein_id=WP_053228785.1] [location=complement(5720769..5721398)] [gbkey=CDS]
MSQQFSYTPGGSPEMDDLKDPHFYEEQRRNYPPGSPYGAPSYPPYFQQPPKKKWLAMLLSLMMPGTGQFYVGMMQRGLFI
MMLVIMDIFAIVLFATRDEGTNVAMITLFSLILPVIYFYNVFEALQLTDIVNRRRFYGESEELTDPLQKLAKGGNLGILL
IAAGGFLFLASSKPVWLESFFKLFGSYFGAIVLIIGGIALFVWETKRKQ
>lcl|NZ_CP026520.1_prot_WP_042234486.1_5159 [locus_tag=PC41400_RS26060] [protein=sigma-70 family RNA polymerase sigma factor] [protein_id=WP_042234486.1] [location=complement(5721720..5722331)] [gbkey=CDS]
MEHLTDEQLVEQINRGQAEAYRVLVDRHKSYVYTLIYRMVNHRETAEDLSQEVFLKLYRSLSHFRGESKFTTWLYRMTVN
LVTDYRRAQRRRPVVPLLDKLKTWLADRRTEPEQQAIRHEESDSLHRMIAQMPEKYKLVMYLYHFKQLSYQEISDVTDLP
LKTIETRLYRGKAMLKEKWLEVYPDARTNAQTSQSAPASRASQ
>lcl|NZ_CP026520.1_prot_WP_042234489.1_5160 [locus_tag=PC41400_RS26065] [protein=MgtC/SapB family protein] [protein_id=WP_042234489.1] [location=complement(5722481..5723173)] [gbkey=CDS]
MNDPWVIGQFDLTIRLILALVLGGLIGFERELSNQAAGFRTHILVCVGSALIMLLSIYGFSAFSKELNVRMDPSRLAAQV
VSGIGFLGAGTIIRNGFSVSGLTTAASLWVVAAIGLAAGAGFYYAAILSTIMVVLSLWILNKIEHKYFGAKKRHLIRVHA
LDRPGNLGKISDILEKNKVQIRKFGVDEEERDERQTVMIYLNVIIPDYHLVPLLLDDIRHLDGVVGVTME
>lcl|NZ_CP026520.1_prot_WP_042234491.1_5161 [locus_tag=PC41400_RS26070] [protein=GNAT family N-acetyltransferase] [protein_id=WP_042234491.1] [location=complement(5723223..5723702)] [gbkey=CDS]
MMRQIPLDDVQAVLELLALQTTAYLVEAGLIGYKDIPPLRDSPLTLRCSKETFTGYYVNDREEGREELAGAIAWEEKGTE
LTITRMMVHSRHFRQGIASALIRHALTTNPQAGRFVVSTGEANLPAVRLYEKFGFHRTGKRTIAPSIALVTFEKIRGIQ
>lcl|NZ_CP026520.1_prot_WP_042234494.1_5162 [gene=gatB] [locus_tag=PC41400_RS26075] [protein=Asp-tRNA(Asn)/Glu-tRNA(Gln) amidotransferase subunit GatB] [protein_id=WP_042234494.1] [location=complement(5723809..5725257)] [gbkey=CDS]
MSATETKYETVIGLEVHVELHTNSKIFCGCSTSFGAPPNTHTCPVCLGHPGVLPVLNRQAVEYAMKAAMALNCEIADVTK
FDRKNYFYPDSPKAYQISQFDKPIGENGWIDIEVGGETKRIGITRLHLEEDAGKLTHVDGGYASLVDFNRVGTPLVEIVS
EPDLRSPEEARAYLEKLRAIMLYCDVSDVKMEEGSLRCDANISIRPVGQEKFGTRAELKNMNSFRGVQRGLEYEEIRQAE
VLDGGGEVVQETRRWDEGQGKTISMRGKEESHDYRYFPDPDLVNLHIDAAWKERVKSSIPELPDARKARYVSKYGLPGYD
AEVITASMKMADFFEESLKYTSDAKVVSNWIMGDLLGYLNAGGLDFGEIRITGQGLGEMIGLIEKGTISNKIAKTVFKEM
IESGKAPQQIVEEKGLVQISDEGAIKSVVEQVVAGNPQSVADYKAGKEKAVGFLVGQVMKETKGKANPGLVNKLIVETLQ
AQ
>lcl|NZ_CP026520.1_prot_WP_042234496.1_5163 [gene=gatA] [locus_tag=PC41400_RS26080] [protein=Asp-tRNA(Asn)/Glu-tRNA(Gln) amidotransferase subunit GatA] [protein_id=WP_042234496.1] [location=complement(5725279..5726736)] [gbkey=CDS]
MSLFELRLQELHTQLLDKKVSVTELVDQSYKRIAEVDGKVKAFLKLDEENARRSAKTLDEQIAAGESRGILFGLPAGIKD
NIVTEGVTTTCASKFLSNYTPVYDATVMRKLKEAQALPVGKLNMDEFAMGGSNENSAFHATRNPWDLDRVPGGSSGGSAA
SVAAGEVYFSLGSDTGGSIRQPAAYCGVVGLKPTYGLVSRFGLVAFASSLDQIGPLTKNVEDSAYVLQSIAGHDPQDSTS
ANVEIPDYVSALTGDVKGLRIAVPKEYLGAGVDPQVKEKIQDALRVLEGLGAVVEEVSLPHSEYAVAAYYLLASSEASSN
LARYDGVRYGIRTDNANNLLEMYHMSRSEGFGPEVKRRIMLGTYALSSGYYDAYYLKAQKVRTLIKQDFDQVFANYDVVI
GPTAPTTAFRLGEQVGDPLTMYLNDILTIPVNLAGIPAISVPCGLASGLPVGLQIIGKALDESTVLRVAHAFEQNTDFHK
QRPQL
>lcl|NZ_CP026520.1_prot_WP_042234499.1_5164 [gene=gatC] [locus_tag=PC41400_RS26085] [protein=Asp-tRNA(Asn)/Glu-tRNA(Gln) amidotransferase subunit GatC] [protein_id=WP_042234499.1] [location=complement(5726803..5727090)] [gbkey=CDS]
MSITIKDVEHVAKLARLDLSAEEKEQFTEQLNAILKYAEQLGKLDTEHVEPTSHAMSVKNVTREDEVRPSLPLDKVMLNA
PEEEDGQFKVPAVME
>lcl|NZ_CP026520.1_prot_WP_009671386.1_5165 [locus_tag=PC41400_RS26090] [protein=hypothetical protein] [protein_id=WP_009671386.1] [location=complement(5727341..5727718)] [gbkey=CDS]
MLSLGARVIIVEDRLEQNLPIGDYAYIIAYDRNNDNLFDYVIRVPKLNKHFFVPASDIELEEVLLRQEVDRLEKEALIDF
ALATRNEELFNRVMNGEPVEIVAEKSKEVQSAEDFIKQVNLRAWI
>lcl|NZ_CP026520.1_prot_WP_042234502.1_5166 [locus_tag=PC41400_RS26095] [protein=Xaa-Pro peptidase family protein] [protein_id=WP_042234502.1] [location=complement(5727942..5729138)] [gbkey=CDS]
MNKTDHPYAERQQRLQKAMQAGGIDAFLVNHHVDLYYFNGSMQNGYLFVPAEGNPLFVVRRSITRAQQESLAEVEEFGSL
RTLKDRLAQRYPGVFASGKEAVIATEFDVLPVQQFQRLEGAFPGVKWKDGSLLVKELRMIKSADEIALIRKAADVIDRTL
EKALAGIREGMPEFELMSLIERENRLQGHLGLMRMRAYNSELITGCVAAGAAAAEPTYFDGPAGGRGLSPASPQSAGSRP
IGRNEPILIDIGCNIEGYVIDQTRTAVIGALPDELQRAYEISEKILKEAEAMMKPGTVAEHLYLQSLDTASAEGLGGHFM
GYGIDQVKFLGHGIGLEIDEFPVLAKGFKYKLEPGMVIAVEPKFTFPGLGVVGIENTYAITETGFEKLTVSREGVLVL
>lcl|NZ_CP026520.1_prot_WP_042234507.1_5167 [locus_tag=PC41400_RS26100] [protein=hypothetical protein] [protein_id=WP_042234507.1] [location=complement(5729859..5730212)] [gbkey=CDS]
MEHVKKPYTGPVYTPKQTGPMPQAVAPMPMPMPMPLPAPMPMPLPYPGPLPSPVIHTPPPMPIVKPVIVVSCKPHYPVCH
PIKPIAKKCRPDNSASILVLFILLVIISSGFGYAKRC
>lcl|NZ_CP026520.1_prot_WP_042234510.1_5168 [locus_tag=PC41400_RS26105] [protein=MBL fold metallo-hydrolase] [protein_id=WP_042234510.1] [location=complement(5730498..5731121)] [gbkey=CDS]
MIRIETFKLGPMETNAYLLTDESGQKGVVIDPGMKPGPLLKRIKDMEIEAILLTHAHLDHIGGVQELRELKKCPVYLHDA
ESDWLGSPKLNGSARWPDVTDPIVVEPAEYALDEGQVLEFLGAKIKVFHTPGHSPGSVSFLFDNHLIGGDVLFRLSVGRT
DLSGGDWNTLLDSIHGKLFELDDDVTVYPGHGPKTTIGYEKQNNPYI
>lcl|NZ_CP026520.1_prot_WP_042234513.1_5169 [locus_tag=PC41400_RS26110] [protein=thioredoxin family protein] [protein_id=WP_042234513.1] [location=complement(5731136..5731702)] [gbkey=CDS]
MSTNLSAVIGKGLTPQQFIDSMQRNQEAFLDWYGKFQWETEDDREFFESLNNRDDLRCVIIAAEWCGDVVRNIPVVFKAL
ENSGMPVEVLIMEEHTDVMDQFLTFGGRSIPVVIFTDTGGHVLGKWGPRPERVQAVMKAFKEGNPDREAPDYNDKIQAVR
AEMLRQYGEGVGYQSVIVSELRELLSTF
>lcl|NZ_CP026520.1_prot_WP_042234515.1_5170 [locus_tag=PC41400_RS26115] [protein=DedA family protein] [protein_id=WP_042234515.1] [location=complement(5731735..5732328)] [gbkey=CDS]
MGEFIHSLLVFFEGLGYWGVMLGLMIEVIPSEIVLSYAGYLVAKGEIGFAGAVFFGTVGGVIAQIFVYWVGRYGGRPILE
RYGKYIFISKHHIDVAETWFNKYGTGVIFTARFIPVVRHAISVPAGLAKMPLGLFTFYTTLAVIPWSMLFVYLGMTLKDN
WENIDDIAGKYTVPILIAAIGLMIIYLVAKKMRAKRK
>lcl|NZ_CP026520.1_prot_WP_042234518.1_5171 [locus_tag=PC41400_RS26120] [protein=cupredoxin domain-containing protein] [protein_id=WP_042234518.1] [location=5732658..5733113] [gbkey=CDS]
MFKSFVLKKYHFVMFLGFLCVAFAAVLFFRNPQILSTAAEPATASERTIQLVTGEFSTKLENGKEIEAYRWDPGTIVVRK
GENIRLSIYGVNGASHPFIIEGLNIKGEVKKGKETVVNFKAEKTGTYRLICMTHTDIAHSGPMIAYINVID
>lcl|NZ_CP026520.1_prot_WP_042234523.1_5172 [locus_tag=PC41400_RS26125] [protein=hypothetical protein] [protein_id=WP_042234523.1] [location=5733237..5733551] [gbkey=CDS]
MKNKTNEKHKPQIPNARGIRRSCSRELYRTVKRLKVYVSPEKIEEAEKLYTRKVAQNIAFIVEHSDNRKVQADWWEEHVA
PEIAVLWDVNADTLSKAFRSAYGG
>lcl|NZ_CP026520.1_prot_WP_042234527.1_5173 [locus_tag=PC41400_RS26130] [protein=polysaccharide deacetylase family protein] [protein_id=WP_042234527.1] [location=5733684..5734616] [gbkey=CDS]
MSKNSGSVKAPLRRTAAACAVCVFFTLAPAPALASGGLHASSGPTVSAPRAEDVNGLRLRSTASPASPISAEKDPHAAAA
KPRDRDYYEQRGEIVWEVPSAGKVIALTFDDGPDPVQTAQILDELKKYGAKATFFVVGKRAERFPELIKREAAEGHEIGN
HTYSHPFLTNNRSTESIRSEIAATQKVIKSINGTAPVLFRPPGGYYHERLVHVSLAEGCLPILWSWHQDTEDWKAPGVNR
VVNKVLNNAREGDIVLFHDHVLGSMNTVDAIRKILPELSKRGYHFVTVSELLKLRGSTSGVSKPSGSKTK
>lcl|NZ_CP026520.1_prot_WP_042234531.1_5174 [locus_tag=PC41400_RS26135] [protein=TSUP family transporter] [protein_id=WP_042234531.1] [location=5734707..5735474] [gbkey=CDS]
MEALSWELLLFLIVTGFLAAFVDSIVGGGGLITVPALLFTGLPPALVLGTNKLAGTMCSLTSMTSFLLSGKIHFRLAAAL
FPLSLAGSMLGALLVHRIPSEFLKPLVVVLLIAVTVYTLLKKNWGAASTFSGLTSRSRFWLAAGAFVIGFYDGFFGPGAG
SFLIFLFLLLGFDFVGASANSKVLNFASNIGALVTFFLLDSVSLAYGIPMGIAMIGGALAGTRFAIRKGASFVKPLFLIM
TTVLIGKQLWTLIMS
>lcl|NZ_CP026520.1_prot_WP_042234534.1_5175 [locus_tag=PC41400_RS26140] [protein=NAD(P)H-binding protein] [protein_id=WP_042234534.1] [location=complement(5735550..5736212)] [gbkey=CDS]
MKKTAIIAGASGLVGRELISVLIADPGYEKIIVLVRTRIEGRDSESKFHQIVTDWNPEQLEQALRGELKQADVFCALGTT
IKKAGTQEQFREVDYEYPMILGNLAKKYGARQFLVVSAMGAKRDSKFFYSRVKGDLEHGLSKLGLRTLRIFRPSLLLGDR
EEEFRFGEKAGAIAAKAFSFLMAGGLRKYRPIPAQRVAAGMAQAAKQKPESIRIYESHQI
>lcl|NZ_CP026520.1_prot_WP_009673274.1_5176 [locus_tag=PC41400_RS26145] [protein=antibiotic biosynthesis monooxygenase] [protein_id=WP_009673274.1] [location=complement(5736408..5736689)] [gbkey=CDS]
MYIAVNTIEVPNPPQMVEMFRKQAPNMKELQGFLGLEVWTDEQAIKVITKWENKEAFEAYTQSEVFKRSHGGHGGQEMKP
KARLSYFEGETLI
>lcl|NZ_CP026520.1_prot_WP_084706679.1_5177 [locus_tag=PC41400_RS26150] [protein=DUF2167 domain-containing protein] [protein_id=WP_084706679.1] [location=5736868..5738055] [gbkey=CDS]
MKLPKLITLVTGAALCVSLWTAPASAEETLNWVEGPKKVELGQIAELDLGKDFVFLNDTDTKKFEQQNGSHVNNREIGSV
FPKDEKQNWAVIFEYEETGHVKDDEKDKIDADKLLESYKTGTEEANKRGDVSNRLFVDGWDVKPFYDNDIHSLSWSLLGH
DSKDEKFVNYNVNVLTRKGVISVVLVTDPKNLAADRKVLTSSILPKLQVKAGERYEEYNSKTDKLAEYGLSGLILGGAGL
AVAKKVGLIGLLLVFLKKFGVILVAALAALWRFLRGKKKKAAAEQAAEAQAAEARAAELTAAGSAGSAADATAQDADPAP
GPSAETAAQQPAGESGRPVVRMLSGAEAPADPTREPEHSRSEAAPASESGTGRAPLTEEDSLRKPDRPEDPDKSK
>lcl|NZ_CP026520.1_prot_WP_042234537.1_5178 [locus_tag=PC41400_RS26155] [protein=amidase domain-containing protein] [protein_id=WP_042234537.1] [location=5738387..5739520] [gbkey=CDS]
MKLKKMVTSLVLISAFTTTSAFAQTSPAGVKDDLYQKAVQKEVENFVKKAYTPYYTINSIQLSTSDYSVKDNQLTARVNI
SLNKTLKAKSVDELPYVKGLKSKLNVLKLRKDTSSTDAEQVVNDRMKDLQEYIGTSTDQNDSFRITAPIVNGSPDLKNAN
LEFLNGLVDWIPADYFIPSNESSMIENGEEFVTEAVTTNKNFLVKSAAEASMVQPLAAVTYDRIAARDYANKWTSNTTNA
SGYDTSKWNPKYAKHTENGGVDCANYVSQAIYTGGIPTDSTWKPESLAWVNTGRNISNGLKQYMVDTKGYFYKTTKTTTP
AGGFISALKYSHVMFVVANDTVTMQFSAHTNDRLKASFANFSTSEYEFYYIKSTYLK
>lcl|NZ_CP026520.1_prot_WP_042234540.1_5179 [locus_tag=PC41400_RS26160] [protein=winged helix-turn-helix domain-containing protein] [protein_id=WP_042234540.1] [location=complement(5739599..5740312)] [gbkey=CDS]
MTIAIYTDDPDMAHVLSYFFNETVCSVHARSELGGLPEPGGGQAILAVGSPDEKDWFRLKRSVKQGTTTYLMIREPMTPL
ETVYAKEIGVREILMDPVTRLRSKAAETPDKLISVLVCPGLYEAAAGSELVYIGEGTYFHPKQFCIKYGSTRIELSEKEA
ALLSFFVENEGIVVTKHTIAEKLWGGYIQPDGISKVIARMKNKLGPARVLISSRQLGGFMYVKEREAAEYAVTRRLK
>lcl|NZ_CP026520.1_prot_WP_042234543.1_5180 [locus_tag=PC41400_RS26165] [protein=copper amine oxidase N-terminal domain-containing protein] [protein_id=WP_042234543.1] [location=complement(5740605..5741666)] [gbkey=CDS]
MNNGMKKGLAVLAVSMAMTTGAAYASPAAVTKADNLVPISAEVSQSVQVTVNNQVLEAEGYMKAGAKEAMIPLRAVAEAI
GMELKWNQEDYSVELSKDRIWTLVKTGEDRYVLNKMYKPLGTAPELVDSKLYVPASFASEILHGTVNAQGSAVSISFNEK
KENVVTKGVVTSVRSADGYKAVQIQGFGTAGLVLNVGDDTVFVTDKGDKLTFDQLALGQIVEAEHSMVMTMSLPPQTPTY
KITVSGYSADAEKLLGTSGTVEEVRTDDNGSQSVTIKGQGLTEQSQSEIVLRLSDATEVVDKEGAKADKKALVKGAKVIG
FYNGVMTKSLPPIGGAVKIVVDVEPEAAAAQAE
>lcl|NZ_CP026520.1_prot_WP_042234546.1_5181 [locus_tag=PC41400_RS26170] [protein=hypothetical protein] [protein_id=WP_042234546.1] [location=complement(5741836..5742120)] [gbkey=CDS]
MTPIHHINYRNEHNEVYCCLRNKVVELDDRQKSDFCSGCQMFAGFAGGKGVECEWEDMRDVPNPMRVTDPMKEFMSNQIR
KIELDDPAVLAHGN
>lcl|NZ_CP026520.1_prot_WP_042234549.1_5182 [locus_tag=PC41400_RS26175] [protein=hypothetical protein] [protein_id=WP_042234549.1] [location=complement(5742233..5742688)] [gbkey=CDS]
MKEFSKTLATMTRPFSFVHKTLSEFGFVRPKTHENQLYVTTFADSITQRVYTLAVPTRLLENGHTEVYLGHTRFDTEEEP
PSRVVEAAICRMKELADYLNQPNKRPHTVETASNGYMATDSYDIRRLGNEMKKMKTNSELKQTGLIPDPIQ
>lcl|NZ_CP026520.1_prot_WP_042234552.1_5183 [locus_tag=PC41400_RS26180] [protein=ABC transporter ATP-binding protein] [protein_id=WP_042234552.1] [location=complement(5742796..5744601)] [gbkey=CDS]
MRNGFGGGLGERNGGKMPKATLKDVSLRRILGLFRLYRGQLTVIVLLALGGAVLGLVPPLVMKEIIDTAIPQQSFRLLAV
FAVLLVALPLLSGMLGVAQNHLNTKVAQGVMRDLRMTLFRNLQRQSMSFFTAARAGEVVQRLTEDIQAVQTVVTSLVVSS
ITQIVIVITTVGILFALDWRLAIVALIILPAFILPVRKVSGVRKRLRGETQKVRADMSAQLGEIFGVSGAMLTRIFTREK
EQEERFGGMNQQVMDLELRLNLIGRWYMMFLGVLGPLGTAVIYMYGGYSVIQGSMSIGGIVAFAAYLGRLYSPVGTLLNL
HVEVATALGVFQRIFDYQDMKAEVEEAEGAAELPPVSGSISVRDVSFAYSPGNTVLSDISFQAEAGEVVALVGPSGAGKS
TLISLMARLHDPSEGTVEIDGFDVKKVSLSSLRRQIAYVTQESFLFHATVRENLLFAKEDATEEELADACRQAYIYDFVQ
SLPQGFDTMVGERGHRLSGGERQRLAIARALLKNPRILILDEATSHLDSESEAFVQAALEQLMKGRTTLVIAHRLSTILH
ADRILVLQKGRIVESGTHKEMMEVDGLYARLYRTQFKRADG
>lcl|NZ_CP026520.1_prot_WP_042234554.1_5184 [gene=yyaC] [locus_tag=PC41400_RS26185] [protein=spore protease YyaC] [protein_id=WP_042234554.1] [location=complement(5744942..5745529)] [gbkey=CDS]
MDISRNEGEAASLAKPPASKASKASKDGYWKKVSGRQLPAFFSEIREEAGAALPCVFVCIGTDRSTGDALGPLVGSMLAE
AGYSVIGTLEHPCDASNLVQRLAEIPAEVPVVGIDACLGQPSSVGLYQVSNRPIFPGKSLGKNLPLVGDYTVAAIVNADG
PRQYAVLQTTSLNRVIGMAREIAAAVQAAFPPVRA
>lcl|NZ_CP026520.1_prot_WP_042234556.1_5185 [locus_tag=PC41400_RS26190] [protein=glycosyltransferase family 2 protein] [protein_id=WP_042234556.1] [location=5745813..5746988] [gbkey=CDS]
MSIAGILALLILLYWLAMLADHLRGARLFRELPPSGPLPDPTPLVSVIIAAKEEETTIRDTVRHLLAQTYPRLELIVVND
RSQDGTGRTLDELKRWSEEERGDIRIPMRVIHVTRLPGGWLGKNHALYQGYRQARGQWLLFTDADIRFHPDTVREAVGFA
LSEHADHLTLTPHMTAPTFWLRSFIRHFLFSLSLFTRPWRANLDGERKGGMGIGAFNLIHRRAYEAIGTHEAIALRPDDD
LQLGARVKLAGLRQRLASGRSRLQVEWYPSMRAAVRGLEKNLFSGFGYKVSVAAAAGAGQLLLFTFPWLGMLLYGDWRSL
VHLASVLIMTALYRKQIRMMAGSPGLDTLAVPASAALLVYVLVRSVGLTLKQGGVYWRGTFYSLSDLKKMK
>lcl|NZ_CP026520.1_prot_WP_009673288.1_5186 [locus_tag=PC41400_RS26195] [protein=copper homeostasis protein CutC] [protein_id=WP_009673288.1] [location=complement(5747666..5748355)] [gbkey=CDS]
MLLEVIATTVGDAIAAEQAGADRLELVTGMKEGGLTPSAGLIEGTVRSVKIPSAVMVRPHSKSFVYDADDVRAMRRDIRL
IRETGAAGIVIGTLTPQNEIDVPVLEALLEEAGGLEVVFHRAFDAVPDQIAAFRQLSGYKQITRILTSGGKPSALDAVDR
MKELVELSRDSHITILAGSGLNAESLESFVKDTGVTEVHFGGAVRGPKGVEDDIAAEKIREIRDILARC
>lcl|NZ_CP026520.1_prot_WP_042234558.1_5187 [locus_tag=PC41400_RS26200] [protein=GNAT family protein] [protein_id=WP_042234558.1] [location=complement(5748459..5749004)] [gbkey=CDS]
MRICQVDEEISLELLELKHAEALFRLTDENREQLGQWLFWAYDTKEVGDSVAFIQSTLHAFAEGKSLAAAIMVRGELAGV
IDYHAMVPAHGRAEIGYWLGKAHQGRGIMTRACRAFIGYGWSEKNLNRIVIMADVENAASRAIPQRLGFTLEGVMREWEV
KESGKRDMALYSILRREWTEN
>lcl|NZ_CP026520.1_prot_WP_042234560.1_5188 [locus_tag=PC41400_RS26205] [protein=hypothetical protein] [protein_id=WP_042234560.1] [location=complement(5749427..5750566)] [gbkey=CDS]
MKNAELYADEWSKLATDRKKEKIESLLAELTLPFIFSGMETFSLNGVTTETIVLNCEGRSFVFVPGCKRAILGWDRGIEG
LYESALEDIASSFESVLDPLTWWRRQLALAVDAQDGNLPYFEEQVMELEQNPPEKPDDDLKAFYSFQGLETYVNENTSPV
RTADLPPMIVERGLSEAGLIHLGQVNRRTNEHWVTSEHFDRVKDYLYKFNTELTTAVYSRCCRFERTKDDPALYDVYLYE
RLTYKDLISQLDAEGFALPTEDQWEYLCGAGRRLLFQFGNQFDEDYRYSHMCEEGENILEKPNMFGLSIAYDPYKYELVT
GPCQVKGGDGGAALCGGEPFIYTMLPLSAYWRETPDKTVPEDGELENGYYFYRRVLIIE
>lcl|NZ_CP026520.1_prot_PC41400_RS26210_5189 [locus_tag=PC41400_RS26210] [protein=IS4 family transposase] [pseudo=true] [partial=3'] [location=5750910..>5752168] [gbkey=CDS]
MKEELQLFSQELTRHLSPSSLSDLAKTIGFVTRSSKFRAQDLVSLCVWLSQKVGATSLNQLCSKLEASTGILISPEGLNQ
RFNSAAVTFHQQILYLLLQQKLLGAASIHSEYARNFSRIRILDSTIFQLPDIFSEHYAGSGGSSHTAGVKIQLEYELQSG
GFLNLYTGHGKENDKTYGTECLNSILPKDLFIRDLGYFDLNGLHTIDENKAYYGLRLKLNTRVYQKNEHPETFGNGTVKK
HSEYIQLSIESIMEQLEPGETREISDVYIGMYQKLPARLVVYRLTEIQMRKREKDQAVREKKKGLTYSEKSKKLSAMNFY
IINITSQVVPGAHLHELYSLRW*AEILFKTWKSFFQIQKCKKMKLKRLQCHLYGQLIGILLCSSTMFQVRETTCHSYIRP
YKKTPRKYRKFCFIISLFS
>lcl|NZ_CP026520.1_prot_WP_042234561.1_5190 [locus_tag=PC41400_RS26215] [protein=hypothetical protein] [protein_id=WP_042234561.1] [location=complement(5752200..5752580)] [gbkey=CDS]
MITVKDIIEFVSDGEKFNFCCSAFLDSFYAAVDQQRIEMIKDEPEAFPDVEHFYYCHVAAMTHKLANDFKLIVPEWVHKE
KYILTEPHYDVFDKNNEILKKYLEDVSPYEFKIRNIFVKENALDRV
>lcl|NZ_CP026520.1_prot_WP_129112538.1_5191 [locus_tag=PC41400_RS26220] [protein=hypothetical protein] [protein_id=WP_129112538.1] [location=complement(5753161..5754165)] [gbkey=CDS]
MAIIQSAEVNKMKNSRRKNRGTLQYHISMKDQEFQLEPYSKRLFDVIKMGLLFAAALYVVLLVFELLTLIGSWSSSSFLG
MQRLNAWICGGVFVLTIVGAFPRLAKRSEHKPEQLRFEDGSLCFRLNGNDVAWRSSDILGVNQHYRRNNLITSWMFVYDV
RTAKGIYRLDLEDYAREVVVANYLLDCNRERNAEKVRKWQSAKQAFRTSHRVTFGTKRRLVVFEQQPPCMTIGHKRLELS
SVRRIVVDPTVEGRGRLNIHVAVIQQNGKEVALSSLRYRAIEDDWDSFLHHLFAIARRANVPVELKTENRRDVSLGPSRN
KTYPIWNGKEGSSE
>lcl|NZ_CP026520.1_prot_WP_042234565.1_5192 [locus_tag=PC41400_RS26225] [protein=hypothetical protein] [protein_id=WP_042234565.1] [location=5754508..5754897] [gbkey=CDS]
MYKEDFYCLCKSYMNRHVRVQTKSGRTYDGMIMGVDPNYVRLKVSGGQGQGYGYGPDMGYGQGYIAGHVNGYGRTGPYGQ
PLDESEERGFFPPAPGFPGYPGFPGYPGPSNETILTLALFDLLTLVLLM
>lcl|NZ_CP026520.1_prot_WP_042234567.1_5193 [gene=hisJ] [locus_tag=PC41400_RS26230] [protein=histidinol-phosphatase HisJ] [protein_id=WP_042234567.1] [location=complement(5755154..5755990)] [gbkey=CDS]
MTFKWDGHTHTKFCYHGNPAEEPAYLDRAIELGFERYTISEHPPLPDKWVDNPRLMAELAMPMEELPDYLAYAEEMKARY
AGRIEVTVGLEMDYLYGREDFSDALLDRYGSRLEDVVVSVHYLPGAGGMRCIDFTADDFREGLLAHYGSMDAVAEEYYNH
VEKAIDWAAGLPMRKRLGHVQLIEKFREALPPISPELIERRLTALLPKLQETGVGIDVNVAGLRVPTCGQTYVPRWFLAA
CVSRGIPLIYGSDTHKPEHVGADWAWFETAFGTSRAES
>lcl|NZ_CP026520.1_prot_WP_042234569.1_5194 [locus_tag=PC41400_RS26235] [protein=FixH family protein] [protein_id=WP_042234569.1] [location=complement(5756152..5756643)] [gbkey=CDS]
MNAHKQENSAGPSGLKLTFRRRRFGTAVLLLVLTAALAGCGSGGGGHEGHGGQGAGQGTIDTLNPIMAKVKLPAEPVKQG
TEAAVEVTVTQDGKPVDDADEVLFEVWKEGAPDDKHEKIKGTHGKDGLYTLKKKFDEPGTYLIISHVTARTMHTMPQVKL
QVE
>lcl|NZ_CP026520.1_prot_WP_009673287.1_5195 [locus_tag=PC41400_RS26240] [protein=KGG domain-containing protein] [protein_id=WP_009673287.1] [location=complement(5756891..5757076)] [gbkey=CDS]
MADKMSREEAGRKGGEETARTHDKDFYQEIGRKGGESRGNDSGRSDGKMSREEAGRKGGRS
>lcl|NZ_CP026520.1_prot_WP_042234571.1_5196 [gene=solA] [locus_tag=PC41400_RS26245] [protein=N-methyl-L-tryptophan oxidase] [protein_id=WP_042234571.1] [location=complement(5757204..5758334)] [gbkey=CDS]
MRTRYDVIIIGAGSMGLSAGYQLASSGVRTLLLDAGNPPHEEGSHHGETRLIRHAYGGASAYVPLALRAQKLWEELERLS
GETLLRTTGVLNVSREETANHRSKIANARRYGVPLEELDAADIHRRWPGFSLPEGWSGLYEPDGGVLLSEACLRVYRRLA
LEAGAELLTDTPVRSVRLDGGEAAVTTPAGTFASGRLIVTAGAAAAGILPGRPLPVTAIRKTVAWFRAGSGLYRSPGFPG
FTVHTGLGEYYGFPDFDGSGVKVGRHDEGIPVSPGEPKHPFGRTAADEEELRAFMEAYLPGASGVLLKGKVCLYERTPDE
DFIIGRLPGHPHVSVAAGFSGHGFKFASVVGEILRDLTVNGRSSLDLTPFSPERFN
>lcl|NZ_CP026520.1_prot_WP_042234573.1_5197 [locus_tag=PC41400_RS26250] [protein=aspartate 1-decarboxylase] [protein_id=WP_042234573.1] [location=complement(5758680..5759072)] [gbkey=CDS]
MFRTMMKSKLHRATVTEANLNYVGSITIDEDLMELADLLPNEKVQIVNLNNGSRMETYTITGPRGSGCVCLNGAAARLVQ
PGDVVIIISYALMSEETAANHVPKVVILEEGNKVAHDRYEEKHSTVPEPS
>lcl|NZ_CP026520.1_prot_WP_042234575.1_5198 [locus_tag=PC41400_RS26255] [protein=YheC/YheD family protein] [protein_id=WP_042234575.1] [location=complement(5759098..5759841)] [gbkey=CDS]
MAVPARSNKWSKYKFMRNAAGLVPHLPATSPMSERSFREMSERYGRVVVKPVGGMQGQGIFHVSCLPDGRYEVQHENRKI
KLAGRRETYAFLRKQIGSQGYIVQQSIDRATVGERAFDLRIIVQRKKNSRRWVVTGRVAKVVGSGYFVSNTSRKGQLLLL
DEALGRSSLHRLSRKALQERIDGVALACAKTLTEHFPRQVIYGIDIGLDKRGNAWIFEGNLYPALSHFVKLKDKTMYRRI
KAYGEGQ
>lcl|NZ_CP026520.1_prot_WP_042234577.1_5199 [locus_tag=PC41400_RS26260] [protein=glycosyltransferase family 4 protein] [protein_id=WP_042234577.1] [location=5760153..5761316] [gbkey=CDS]
MSHYQVIWNGPVLGDNGIGTAGREYVLALHRLGVHVKVTPARRRSSLSTPSKRLMLALARRPLVRKKPKVLVYHRLPHTL
NIAQARKKYDYLILNAVWETTKIPRNWLPNINRFDAVCVPSVQNRIAMRRSGVRVPVYIVPHGVDSRAFNPRNKKLAVTK
KGAFLFVSVFTFQHRKNPETLLRAFWEEFSARDRAALLIKTSGFSSKESGAWIRGRIAAYKKELGIRHKTAPLHLLTGSA
SPRTVRGVYTAGDAFVLPTRGEGVGLPFLESLASGTPVIATGWGGHMDFVTRANSFLVPYKLKPPAAGMKKAISRSFRHL
FAQKGQLWAEPDLDSLKKQMRHAYEHPALCRQKGLRGRRDMSRWTWARAGHAMKNAIEKTIASKRRR
>lcl|NZ_CP026520.1_prot_WP_009673270.1_5200 [gene=gcvH] [locus_tag=PC41400_RS26265] [protein=glycine cleavage system protein GcvH] [protein_id=WP_009673270.1] [location=complement(5761533..5761925)] [gbkey=CDS]
MSEVKSGLLYSKEHEWVEVLSDTKIRVGMTPFAQEQLGDIVFVELPKTGASVKADDMLGSVESVKTVSDLISPVIGTVAR
INDTLEESPELVNTDPYGAGWVAEIDISGKEALDGLMTADQYQAYIVSEQ
>lcl|NZ_CP026520.1_prot_WP_042234580.1_5201 [locus_tag=PC41400_RS26270] [protein=hypothetical protein] [protein_id=WP_042234580.1] [location=5762113..5763507] [gbkey=CDS]
MYAATRLNATRFGSALAAGALLLTLWSGQAHGQGAGARAAAPLTAEQTTLLSEEYRQFLKDQFGYTLPSVPTKGDFVKAL
AAVLNLEPSGKPNPFQDIKESSPYYKSALALYEKGILTQDRLDGDQPLTQSTAVYIAVKAAGLKELAYTYPAEKAARSLA
KLHTDDEISQLTPQAAQELAAAADTELLPAGLHEAFQRNEPASASYAEILLGRVLELQGLYKHELGRVSDPDILRQVRDA
WNVQNLIQQPDLQNIVDQALQADLITGYNVKDDRYTPRFDSARSLTYGHSEIDHALQLIGLLRSENIDARVQLEPKTSAF
IYLKEWGEPVQTDDYKVVKIDNGNYIAYAKEYDLSFEFATAQQKDRFQEIVLKYAKKNSEDQSGLIAHSWWQPLYYSRTE
LPDYKVIANNKITKGHYYAQTFSLADKPEAIAEGIRKISPATKVESYRFWVDEPFYNYLIGEYK
>lcl|NZ_CP026520.1_prot_WP_084706681.1_5202 [locus_tag=PC41400_RS26275] [protein=NTTRR-F1 domain] [protein_id=WP_084706681.1] [location=complement(5763661..5767359)] [gbkey=CDS]
MTVLFNNLIQNGGFELGSTDAWGVYNAVAVSTNPKVGAYACLLTGGNVGAGIVQAVFALPGESYSISASFAKTGTGTSPQ
VNVALAYYDSSYLFISYGIASTIQQGTLPDVATGSWKMLYEVTGEVPANAAYASFLVTLSPLQNSSNVLMDEVFVVNASA
SASGVTGATGATGPAGPTGPSGPAGSTGATGADGATGANGADGATGATGADGATGATGADGATGATGATGAAGVTGATGA
TGADGATGATGADGATGATGADGATGATGADGATGATGADGATGATGADGATGATGADGATGATGADGATGATGADGATG
ATGADGATGATGPTGATGATGAEGATGATGADGATGATGADGATGATGADGATGATGADGATGATGPTGATGATGADGAT
GATGADGATGATGADGATGATGSTGATGATGADGATGATGADGATGATGADGATGATGSDGVTGATGADGATGATGADGA
TGATGADGATGATGADGGTGATGADGTIGATGPTGATGATGADGVTGATGADGATGATGADGATGATGADGATGATGADG
ATGAKGADGATGAAGADGATGAAGADGATGAAGADGATGATGADGVTGATGADGATGVTGADGATGVTGVDGATGATGAT
GPTGATGATGADGATGATGADGVTGATGADGARGATGADGATGATGSDGATGATGADGATGATGADGATGATGADGATGA
TGADGATGATGADGATGVTGADGATGATGADGATGETGADGVTGATGADGATGATGPTGATGVTGADGATGVTGADGATG
ATGADGVIGATGADGATGATGVDGATGATGADGATGATGADGATGATGVDGATGATGADGATGATGADGVTGATGADGAT
GATGTDGATGATGADGATGATGADGTTGATGADGTTGATGADGTTGATGADGATGATGPTGATGATGNDGATGATGADGV
TGATGADGVTGATGADGVTGATGADGATGATGADGATGATGADGATGATGADGATGATGADGVTGATGADGATGATGADG
VTGATGPTGVTGATGADGVTGATGADGATGVTGADGATGPTGPTGATGADGTAGTGLTSFASRGNTGGTSVAAGGLVPFN
TAGVQYGTAITFSPTEPTTTFLINELGYYRVSFLIYTETLSLLSSVAASFSGTAVPQPSDSVFSLALAGGVLTGDLLFQA
ETTGSLTVSVSGLTLTLSGTGTNAQITIEKLD
>lcl|NZ_CP026520.1_prot_WP_042234583.1_5203 [locus_tag=PC41400_RS26280] [protein=glycosyltransferase family 2 protein] [protein_id=WP_042234583.1] [location=complement(5767438..5768283)] [gbkey=CDS]
MGALTSIIILTRNGLKHTRACVESVQKYTSLPYEFIFVDNGSTDGTLAYLSTVRNATVLANSVNKGFAGGCNQGIRAAKG
DYLLLLNNDVVVTENWLEGLRGWLGWDRTIGIVGPVSCNVAPIQRIQPDPYQTMDEMQRFARKWREQNIGIGFYPHRLIG
FCMLFHKKLIETIGGFDERFYPGGYEDDDFCLRARLSGKILWAACDVYVHHHGHATFTANGLAWNEQGLYNAEIFRNKWN
LEVSAVELEKTGYNPSLIVEKVKTFCPEKHFVHLGEEEGLT
>lcl|NZ_CP026520.1_prot_WP_042234585.1_5204 [locus_tag=PC41400_RS26285] [protein=3-ketoacyl-ACP reductase] [protein_id=WP_042234585.1] [location=5768510..5769229] [gbkey=CDS]
MELKNKTAIITGAGKGIGKAAAAALAKEGVNLGLLARSTADLEALQKELSGTYGVNVFYASADISKPSEVQQAISSLTGE
LGHVDILINNAGIAQFGTVADMDPEQWERIIQVNLMGTYYVTRAVLPGMIQQNSGNIINVASTAGERGFATGSAYCASKF
GVLGLTEALMQEVRKSNIRVTALTPSTVNTDLASNAGLKIGDEDRMMQPEDVAELITTTLKLPPRVFIKTAGIWTTNPQ
>lcl|NZ_CP026520.1_prot_WP_042234587.1_5205 [locus_tag=PC41400_RS26290] [protein=cupin domain-containing protein] [protein_id=WP_042234587.1] [location=5769447..5769947] [gbkey=CDS]
MSAADVDVKTFFFKEDGDIPNHPHLPVLLYSGVFSENAEQTERLFNRHNWLNSWTNGVFDYHHYHSNAHEVLGVISGSAV
VRLGGEHGESVELRTGDVVVLPAGTGHKKLSGSPDFKIVGAYPDGMEPDLHKESATPAIREKIRSVPVPDTDPVFGEEGP
VLRLWK
>lcl|NZ_CP026520.1_prot_WP_042234589.1_5206 [locus_tag=PC41400_RS26295] [protein=UvrD-helicase domain-containing protein] [protein_id=WP_042234589.1] [location=complement(5770233..5772569)] [gbkey=CDS]
MSQAPFYTVPPGTGHTNIPQAVMASAHTSRELVPDGERDAFYFRRLEAQGIALNEPQIRAVRHFEGPLLTLAGAGSGKTS
VLVCQTGYLLAVRGIRAASILLVTFSSKAAAEMRERIAALPGITAQEASRLQARTFHSFFLRVIRSQGVQEEIFNETRRQ
HILLKQIMRELGIPDTYQPETLLTLLSSYKMNRTELQGLPQQTKEEKEIRSILIRYEEWKRANAKIDFDDVLLRAYRMLK
ENESLLRSLQNRFQYVMVDEFQDTNGIQYELIRMIASPQNHLMVVGDDDQTIYSFNGARSEFILHFEKEFPSAKVITLAI
NYRSGASIVGLGNEIIRHNKMRRVKTLKAANAGSARPQYIRPLTTDEEAEIVLTRIRQDVERGKRSYGDYAVLYRSASNN
RALLEQLVLREIPYIDYGDGQLLYEHWLIKPVLDHMRLSLNRRDFEAMEGVLPSLYINREHGMAYIRQQEARQPKKGPLV
HLLSMPDKKDFQIEKIQTRLELIRELKKLKPIDAIRKMRVHFYDAYVETDGRKQPTQHKEMLKEMLDELETSAERFRTTA
EFVLFVDDVLSRSRQSGRHLSEEQGNRVALMTIHRSKGLEFPVVVLIGACEGSLPHSSVLDPKAVDGAADLEEERRLAYV
AVTRAREELIVSSPSLYRGRKAGVSRFVLSAFTGRSGDTPASGRRAQPSSGKRTVTAVSRTAAASGRAPERSAPAAPVRT
ETVPAWLCTGPGCNAWVRIAAGEPAAGKASGAPVKACPLCGAPMTAGRRETAVRTAGR
>lcl|NZ_CP026520.1_prot_WP_042234591.1_5207 [locus_tag=PC41400_RS26300] [protein=cell wall hydrolase] [protein_id=WP_042234591.1] [location=5773042..5773683] [gbkey=CDS]
MKKMPILLSAAACAIGLVSTAPSASAMSLAKGTQSEHVLDLQERLDALGYFDAGVTGYYGMKTAESVKKFQRAYGLPIDG
EADSKTLGKLEKAAVPKQNALEQMARIIYSEARGESYQGQVAIGAVVINRVQSNQFPDSIPGVIHQPGQFSAVRDGQYWL
KPNETAYKAARAALNGSDPTGGALFYYNPDVATSSWSKSRPKTAVIGNHVFTD
>lcl|NZ_CP026520.1_prot_WP_042234593.1_5208 [locus_tag=PC41400_RS26305] [protein=hypothetical protein] [protein_id=WP_042234593.1] [location=complement(5773788..5775206)] [gbkey=CDS]
MNGPKPEWYDKLKSGPFTRKTFTMDDIYRIEQKTAESAPGGRVRKLTIPKRRRTLPVLAAVFLCVAAAAFLFRGELEPAV
TKAWSGFVKPTEPPQATVPPLQQETGTFYLKGDAAVVSEPKAFGDEETETTVNSENAVTVLEKRNGFVKIGEKRWIPEWY
LNKDAEKRLKEVRLSQFLRERPVVTMTAPYYGVIMQDVNFSLYPGEPKPSGGVLKYGRVVEVIRDYRDWVCVNVFSYDGR
KYVGDKWVPKENVGVYDSSVAQEGLLKERVEALSPSGTPAYLHPGARIVIEGEADGFYRIRSQSTYTAKINKSDFLPNPF
LGLERKEDTDPQWYFTQEEMLNRYEDFTKTRTDDRLKGLEPIDIFRYYVMASEKGDWETVYALLIKGSDYATPGREEYLS
DIGREPDADKQAKQKWDVLRKDYTLYQEISGDSAYIVMTKRGSGSPSPEDKQAFSLTRNKAGIWKVSWLPMQ
>lcl|NZ_CP026520.1_prot_WP_042234701.1_5209 [locus_tag=PC41400_RS26310] [protein=sigma-70 family RNA polymerase sigma factor] [protein_id=WP_042234701.1] [location=complement(5775193..5775678)] [gbkey=CDS]
MLDYGQDVWNYAFFLTGRREMADDISQDVFLSVFRHLNTFEGKASVKTWLFAITRNTARNHMRTAFMRRVILVGRARPNS
SHPSAESEVMEAAFTDEIWEIVMKLPAKFREVLVLDAKYDMTMGEIALVLGVSEGTVKSRLSRARRKMSEAWKEESAYER
A
>lcl|NZ_CP026520.1_prot_WP_042234595.1_5210 [locus_tag=PC41400_RS26315] [protein=hypothetical protein] [protein_id=WP_042234595.1] [location=complement(5775951..5776496)] [gbkey=CDS]
MQEGIERDFNPPISKSNSNKMRAFIRRSVFVLPLVAALSACSVTHSVETDTIRVAAKSMKKNLKGYKGLYCTFTRPDVTC
RVDWKGEPDPETHAAVLAAVKAFVTPEHMREIARSVKWNLEISEFHLLLNTDDNKADAEHAYSARYYKTSDASDDSEANI
DAYKTWTEDGAQPVPLENGSD
>lcl|NZ_CP026520.1_prot_WP_042234597.1_5211 [gene=nfsA] [locus_tag=PC41400_RS26320] [protein=oxygen-insensitive NADPH nitroreductase] [protein_id=WP_042234597.1] [location=5776749..5777483] [gbkey=CDS]
MNSVIDLLTRHRSIRKFKPDPIPGDQLQAIIASAQMASTSSSVQAYTIIAATDPALKKELARLAGNQAYVEQCGLFLVFC
ADLNRLRKATEQSGEVFHQNTESFLVASMDAALAAQNAAIAAESLGLGICYIGGIRNNPDGVAEALKLPELVYPVFGMCV
GVPDQEPSLRPRLPQTAVLHENGYQSDRSDEGVRRYDETTRKYYEERTGGKQSITWSRSMADKYRQPVRAHLKPFLEKQG
FRLE
>lcl|NZ_CP026520.1_prot_WP_042234598.1_5212 [locus_tag=PC41400_RS26325] [protein=GntR family transcriptional regulator] [protein_id=WP_042234598.1] [location=complement(5777548..5778276)] [gbkey=CDS]
MNLNPSTPQPLYMQIRQMLKNDIQQGRYKPDEQIPTEAELCDTYNVSRITIRKAIEELVRDGTLTRIPRRGTFVASNKFH
NELLSVSGFSEFSHQLGMIPNSRILRSEVIQASEDVAGHLQIEEGSPVLELERLMYVDDRPLFYDIAHYSLTRFPDLEKK
IAMNESTYKILSEDYQTEIVSNDKIIDVIGATKDYAKLLECDIGANLFRILKIAFDANDEPVHLSTFMCETNRVNLTVHR
AK
>lcl|NZ_CP026520.1_prot_WP_042234601.1_5213 [gene=frlD] [locus_tag=PC41400_RS26330] [protein=fructoselysine 6-kinase] [protein_id=WP_042234601.1] [location=complement(5778342..5779124)] [gbkey=CDS]
MRIVTIGDNCMDVYQASGKAYPGGNPVNVAVYLTELGAQTAYIGWVGTDMYGEIMIRAIKDKGVDTSRISKKAGKTAVTQ
VEMIGNDRKFGDYDEGVMAQFSLTPEEISFAGNYQLVHSGIWGHADKYFAAFKEKGLVTSFDFSDQLGDELVQTLTPHVD
YPFFSYTRDDDYIRRLLTEVKQRGSQIAVATLGENGSLAYDGERFYTCGVGKVNVADTMGAGDSFIAGFIHGRLNGLRVE
ACLEQGASTAAKTIGYFGAW
>lcl|NZ_CP026520.1_prot_WP_042234603.1_5214 [locus_tag=PC41400_RS26335] [protein=SIS domain-containing protein] [protein_id=WP_042234603.1] [location=complement(5779185..5780198)] [gbkey=CDS]
MLNFDEQLFLKLVEQEGLAFRGQIEEIVDGICKKGYSNIFLIGAGGTIAMMYPYEYILKSNSTIDVHAEIAAEFMVMNNR
HFSKDSVCIFTSVSGTTQETVAAAEFCKERGATTIALVAEPDTPLTKIVDHCITTGSEKHSFDTFFMLLHMVVFRFMYNN
NEFPQYDQFVKEVSLLPRAILDAVKTFDPKAEAFAKEHKETGYHMMVGSGNLWGNTYSYAMCILEEMQWIHAKSIHAAEF
FHGTLELVVEDTSVILLKGEDETRPLMDRVERFAEKITKKLTLIDTKEFKMEGIGEEFRKHFSVNINWSVLSRISVHLER
ERNHPLTLRRYYRQMEY
>lcl|NZ_CP026520.1_prot_WP_042234606.1_5215 [locus_tag=PC41400_RS26340] [protein=amino acid permease] [protein_id=WP_042234606.1] [location=complement(5780303..5781646)] [gbkey=CDS]
MSNGTLTRKLGFWSALAIAVGTTVGSGIFVSSGDVAKAAGLPSISILAWIIGGVIAIPQVMVLAELSTAYPQNGSGYVYL
NKAGWRPLAFLYGWATFWALDPPSISIMALAIVAYLASFFPFFAGFAGKLLGVAIILIITSIHYRSVKGGGSFQVIITAV
KIIPFLIVIVLGLMYMNFDNFAYTPAAGAGSSSLIGGVSATTWAYTGMAAICFMAGEFKNPGKVLPRALISSVLIILGLY
TLLAVCVTGLMPFDKLMGSSSAVSDAVKYIPGLSGIASSFVAVTAIIVILGSLSSCIMFQPRLEYAMAKDGLFFQRFGKV
HPKFETPSFSIIAQVTFACILVFFSNLTELLGYFTLIQLVINILDFAAVYKCRKRDDYNPIYRMPMWRVTTILAIAGAAW
LAWGTFTWAPIQGMIAAAIVIATGLPVYYYWEKKYGSKKSGDSPIVA
>lcl|NZ_CP026520.1_prot_WP_042234607.1_5216 [locus_tag=PC41400_RS26345] [protein=TIM barrel protein] [protein_id=WP_042234607.1] [location=complement(5781663..5782490)] [gbkey=CDS]
MNLAGMNITYKHFPFEHFLDSMSELGIGHIELWAGEPHLYVYRNVLGNLRQIRRQLKSREMQVVCYTPEQCVYPFNIAAS
DAHLRQKSVEYFIDNLYAALKLGTDMMLMTSGIGDFPVSREESWKYATDSIYQVSRVAEKEGVTLALEPLTRFESNLLID
VEGILRVMRDVNSPSLKGMIDTVAMQLAGETPEDYFTKLSELCHFHLIDGDGSSDAHLALDDGVLNWREYITTLQSRGYK
GACTLEIMGSGYYQSPHEALKKSIRKIRDSEVYSS
>lcl|NZ_CP026520.1_prot_WP_042234609.1_5217 [locus_tag=PC41400_RS26350] [protein=hypothetical protein] [protein_id=WP_042234609.1] [location=5782862..5783083] [gbkey=CDS]
MSENNDKQSRNRDGKNGINAGEAVGTVGGAAAGAAVGSILGPLGTLAGGVAGAALGNKAGEGMGDNDKQSGQK
>lcl|NZ_CP026520.1_prot_WP_042234611.1_5218 [gene=map] [locus_tag=PC41400_RS26355] [protein=type I methionyl aminopeptidase] [protein_id=WP_042234611.1] [location=complement(5783425..5784171)] [gbkey=CDS]
MIILKSPREIEEMKPASQIIADCYREVAKLIEPGITTLEINDFVARHITKLGGKQFTKGYNGFPAETCTSVNDVVAHGIP
SNRALMDGDLLKLDIVVEYGGWFGDSCHSFAVGNIRPEAQKLMKVAKECLDLGIARAIPGGRLGDITSAIQQHAEANGYS
LVRDLLAHGIGRSLHEEPSYEHIGVAGKGIRLKEGMVFTIEPMLNEGTYRITIDDDQWTARTADGKLSAQYEHTIAVTSD
GPLILTAL
>lcl|NZ_CP026520.1_prot_WP_042234612.1_5219 [locus_tag=PC41400_RS26360] [protein=TetR/AcrR family transcriptional regulator] [protein_id=WP_042234612.1] [location=complement(5784198..5784968)] [gbkey=CDS]
MTPRTKEQNEEIRLRRLAQIRKAAADVFLNKGPLLEIRDVAAQAGLGYGTVYHYYSNKGDLLHDLLWDALERAGGWLEAP
LEALPKAPPEVSLEVPRASASGKAPVGGHLGLAAAADTGNSAAADSRSGSRETAAAAEFGPVAAAGVRLLQLWAEAPALY
LLHKQAGEGFASLPEARSAPLSAAFRREVLAPLAALLETGRAADGAAASGENAAEPPYRLQRAEMLLAALVGCASLSLRR
GKLYEEAGDIARFLKL
>lcl|NZ_CP026520.1_prot_WP_042234613.1_5220 [locus_tag=PC41400_RS26365] [protein=RsmD family RNA methyltransferase] [protein_id=WP_042234613.1] [location=complement(5785304..5786260)] [gbkey=CDS]
MNSSADGRSVPSFVYSFACHEDEKELCMLELRTLLGVRPQDGWAESPLELDPSRSPFIRHRLEVRLQAASLDELAERAKT
LDLNGDTFKLLYVEAEGKASYDERRAIERRIGQHVRGKADMKQPRHLLGAAYAGGRWVLGPCRRSEAVWLAHNSKPRQYS
TALSTRVARAAVNIAVPDPAGIRAVDPCCGIGTVLVEALSMGIDIVGRDLNPLAVRGARENLAHFGYPDVVKISDIRDLD
GAYDALLLDMPYNLCSVISTEEQLEMLRSARRLSPRAVVISAEPVDAALERAGFAVKDRCVVRKGSFERQIIVCERLR
>lcl|NZ_CP026520.1_prot_WP_042234615.1_5221 [gene=deoD] [locus_tag=PC41400_RS26370] [protein=purine-nucleoside phosphorylase] [protein_id=WP_042234615.1] [location=complement(5786639..5787349)] [gbkey=CDS]
MSVHIGAKENEIAERILLPGDPLRAKFIAENFLEDAVCYNEVRGMYGFTGTYKGQRVSVQGTGMGVPTISIYAHELLESY
GVKTLVRVGTCGAFHEDVKVRDLVLAMTSHTDSAVNRIRFNGLDFAPAADFGLLKSAYEAAESAGMPVHVGSVFTSDTFY
RDDEGATAALLTKYGTLAVEMETTALYTLAAKHKARALSILTVSDHLVTGEETTAEERQTTFRQMIEVALDTVIQY
>lcl|NZ_CP026520.1_prot_WP_042234616.1_5222 [gene=msrA] [locus_tag=PC41400_RS26375] [protein=peptide-methionine (S)-S-oxide reductase MsrA] [protein_id=WP_042234616.1] [location=complement(5787457..5788002)] [gbkey=CDS]
MTETEKQGATYELATFAGGCFWCMVTPFEELPGIHGIVSGYTGGSEVNPTYEQVKAGETGHAEAVQIRFDPDVFPYEKLL
ELFWQQVDPTDAFGQFQDRGLSYRAAIFTHTEKQRELAEQSKRELAQSGRFDKPIVTEIVPAGPFYEAEEYHQNYHRKNP
KHYKEDRAQSGRDEFIGSHWK
>lcl|NZ_CP026520.1_prot_WP_042234617.1_5223 [locus_tag=PC41400_RS26380] [protein=hypothetical protein] [protein_id=WP_042234617.1] [location=complement(5788178..5788489)] [gbkey=CDS]
MGEMDGQTIQVLNASMIHDVENGYIGKVEFKLAEGKTAYEVALFSKKGKEWMYSLNFTAASGTEEEIVAIEQQLEEDDDL
FDFLVNTALSRMEGDSSGDVEGE
>lcl|NZ_CP026520.1_prot_WP_042234618.1_5224 [locus_tag=PC41400_RS26385] [protein=LLM class flavin-dependent oxidoreductase] [protein_id=WP_042234618.1] [location=complement(5788659..5789714)] [gbkey=CDS]
MKPLPLRLSVLDLVPVMQKADSHTALGQAVTLARTAEKLGYTRYWVSEHHDMKQLASACPEVLLAHIGAHTEHIRIGSGA
VLLPHYQPVKVAEAFHLLATLHPGRIDLGIGRAPGGSAHVTLALSGNFLDNVRKMPDKIRDLTALIEGEFTVEGERVTAR
PVPPDSPELWLLGTNAKSAAYAAEYGTGYVFGRFMSDNDGAAVLEEYRSAFRPSRLLAEPKAIVAVGAVCAETEELAREL
AAEGAGWLRSAIPAAEEAAGSEAVQAQVRHRPQGASSAADEAAGGRQPEKADAQGRRTLIGTAGQIKEQLHELQKLYGAD
EFLIVTQVADYAQRLKSYELLASCIIGAGRR
>lcl|NZ_CP026520.1_prot_WP_042234619.1_5225 [locus_tag=PC41400_RS26390] [protein=glucose 1-dehydrogenase] [protein_id=WP_042234619.1] [location=complement(5789740..5790513)] [gbkey=CDS]
MYLPSFQLKGKTALVTGAGRGIGRALAIGLAEAGADVVLLSRTQSDLDAVAAEIAEATGRKAYAIAADAGTREGVTEAVD
RAVREAGRLDILVNNAGMNIRTPALEVTDEEWDAIVQTNLKSAFLMSQTVGRHMKEHGGGRIVNVASVAGHMALRTGVVY
GSTKAALIQMTKILALEWAKYGILVNAIGPWYFPTPLTEKLLADESFVAEVTGRTPLKRVGRLEELVGPVVFLSSDAASY
VTGQTLYVDGGMTIYGF
>lcl|NZ_CP026520.1_prot_WP_042234620.1_5226 [locus_tag=PC41400_RS26395] [protein=phytase] [protein_id=WP_042234620.1] [location=complement(5790768..5792156)] [gbkey=CDS]
MKLFKSFSVLLIAASFYSQAAWASVEEGYEPLRDMLEQAGARVTWNQSTLTADFALQSGLEGSVTIGNDGYKLNGTDGKL
NGHVKLIGDKTYVPPGFVELVLSKLRSGGGDQTAVGEALARVTADGETEAVDTGEDAADDPAVWVHPDDPAKSKLLATNK
GGGILVYNLDGKQVQSYKLGKMNNIDVRYGYELNGKRMDIAAATNRTSNTVDVFSINPETGALTNIAAKPIKSDMGEVYG
FSLYHSLKTGKYYALVLGKEGEFEQIELIPGPAGVEGKLVRHFKLASQSEGIVADDEYGTIYIAEEDAAIWKIGAEPDSG
LKPYAKVDEAGKGHLTADIEGLALYYGADGKGYLMASSQGSSTYAVYNRQGTNRYLGSFAVQDGLIDGTTGTDGIDVVGF
GLGPAYPNGIFVAQDDENTDNGKKRNQNFKIVSWDKLAKSFHAPLRIDNDIDPRKLSSRKNG
>lcl|NZ_CP026520.1_prot_WP_042234621.1_5227 [locus_tag=PC41400_RS26400] [protein=iron-siderophore ABC transporter substrate-binding protein] [protein_id=WP_042234621.1] [location=5792418..5793410] [gbkey=CDS]
MNRPKRSLSTLALLIFTIFALVVSGCGKPGTAPAASTTLNTASPAAEADTARKVKHVMGETEIKGTPQRVVILTNEGTEA
LLELGVKPVGAVKSGVGKTWYPHIKDAMQGVTDLGEEVQPNLELIVTLKPDLIIGNKVRHEKIYGQLQKIAPTVFAETLT
GQWKENFKLYAEALNKKTEGEKALAAYDKHVADVKAKFGSKTSSKVSIVRFLPGAVRIYQNDTFSGVILKDLGFARPAAQ
DKNAFMEVITKERMVDMDGDILFYFNSDYDDSKGGTKTQQEWMSDPLYKNLNAAKTNNAFQIDEIIWNTSGGIKAANLML
DDLVKYADKL
>lcl|NZ_CP026520.1_prot_WP_042234623.1_5228 [gene=ant(6)] [locus_tag=PC41400_RS26405] [protein=aminoglycoside 6-adenylyltransferase] [protein_id=WP_042234623.1] [location=5793729..5794583] [gbkey=CDS]
MRSEQEMMNILVDFARNDDRIRLATLEGSRTNKNIPPDTYQDYDISYFVTDMDSFKENDQWLETFGKRVMLQKPEDMELF
PSELGNWFSYLILFGDGNKVDLTLIPINEAEDYFTNSDGLVEVLLDKDTLIKNEVIADDRQYWIKKPTAREFDDCCNEFW
MVSTYVVKGLARKEILFAIDHLNEIARPNLLRMMAWQIGSEQGYTFSVGKNYKFINRYLPNEDWESLLSTYSENGYPEMW
QSLLTCYELFRKCSKAVAESLDYTYPDYDETITAYTERIYTSLT
>lcl|NZ_CP026520.1_prot_WP_042234625.1_5229 [locus_tag=PC41400_RS26410] [protein=FTR1 family iron permease] [protein_id=WP_042234625.1] [location=complement(5794651..5796501)] [gbkey=CDS]
MATPFFKRKKRSLSVAAGRSVFKIGLAALLLLISVTAAGTGYSPAFAAGESKASGADGMLPVVGGALVEAGQGSWAKAAE
HLAEADKLWQANAAKSSAAAADVNSALAHAKRTVAEGDKDPKAATAALSELAKAVNKYAKEGRQDAQTGQSGQDTAVSLL
PVMKQLAASIQSGDLTQAYNDYKTLNNGWLKVEQAIRSDNFTVYSGMETQMSMIRVALQAEPKQPEQAASETAALIQLVN
DYTAGKIVSKAPEPADMSVAELIGILDQASSDIAAGHHTEAAGQMQAFIQKWPSAEGQVRIRSAAVYTQIENQMTEASRY
LMSVPPAGDKAKEVIAAMREQLQPMTGEKRYTAVDAGLILLREGLEAILVISALLAYLKRTGNGDKRRWIWSGVGSGLLL
SGAMAFLLSYMISQAASGSTREFIEGGTGLLSVVLMLTVGNWLHAKSNVKAWNSYIDRQMGSALAGGSLWSLFAVSLLAI
LREGAETTIFYVGMAPSIAPYQLFLGIGVTLLLLIGIGFAIIRLSAKLPIRPFFITASALIYYLVFRFLGESIHSLQVAS
VVPAHSASGLPSVDWLGIYPTWETLLPQLALLAYIAAKLLRSRSGKASSEPALTKR
>lcl|NZ_CP026520.1_prot_WP_042234627.1_5230 [gene=efeB] [locus_tag=PC41400_RS26415] [protein=iron uptake transporter deferrochelatase/peroxidase subunit] [protein_id=WP_042234627.1] [location=complement(5796482..5797780)] [gbkey=CDS]
MSGNDNDQKKGFFDKPISRRDILKLAGVGGAGLLLGAGGASALVAAQASKTEKAFSASGAETDPQGAVPFYGGHQAGIVT
PVQDFICFAAFDVTASSLAEVRKLFQAWTAAAANMAQGIMVGQDDGNELLPPSDTGEAAGLTPSRTTITFGAGTSFFDGR
FGLAGKKPEALQDLPAFGGDELRPEWCGGDLCVQVCANDMQIAFHAIRNLARIARGKAVLRWTQEGFQRTGYSDPAKATP
RNLMGFKDGTGNPDVKDEQEMKKVVWADSADGSGWIHGGSYMVVRRIRMRIEVWDRSTLADQEATFGRHRNSGAPLGASG
EFDAPDMEKKDASGKPVIPATSHFKLARGDGSAKILRRSYSYSTGIDHKTGQLDAGLLFVCYQRDPRKQFVPIQKALGKS
DKLNEYIVHIGSAVFACFPGVKEGGYIGDTLF
>lcl|NZ_CP026520.1_prot_WP_042234629.1_5231 [locus_tag=PC41400_RS26420] [protein=imelysin family protein] [protein_id=WP_042234629.1] [location=complement(5797777..5798664)] [gbkey=CDS]
MKIRYVALFAACTVLFTGCGTKNNENSGASATPDAKATPSAAAATPAPAGDESWKPILDAYRTYATEQSDLFVKETEKFV
KAVKSGNVEEAKKLYAPTRMYYERIEPIAEALGDLDPNIDARENDVEEKDWRGFHRIEQALWQKNTTKGQEEHADRLLND
AKQLRALIETVEVDSGLLVTGAVELLNEVSASKVTGEEERYSHTDLYDFVANVEGAKKIYTLLQPKLKSKDASLEETIGA
KFKDLEQALATYKEGDGYVSYEKLKQDDTKKLSRLLDALAEPLSQMGTILGAAAK
>lcl|NZ_CP026520.1_prot_WP_042234631.1_5232 [locus_tag=PC41400_RS26425] [protein=VOC family protein] [protein_id=WP_042234631.1] [location=5798899..5799288] [gbkey=CDS]
MLHHVEINVSDLNESAPFWEWLLSQLGYETFQRWEKGISWKCGDTYLVFVQTEERFLDLPYHRRRTGLNHLAFHAKSREQ
VDGLTEALAARGVRVLYREQHPHAGGPRHYAVYFEDPDRIKVEVVAPDV
>lcl|NZ_CP026520.1_prot_WP_042234634.1_5233 [locus_tag=PC41400_RS26430] [protein=AAA family ATPase] [protein_id=WP_042234634.1] [location=complement(5799372..5800430)] [gbkey=CDS]
MNTENRNIQDTELKKEVVTTGLTLGKFAPLHRGHQLMIETAIEENDDVIVLIYDSPDVTDVPLPVRAAWIRTLYPGVEVI
EAWDGPAEKGYTPEIQRMHEEHILAMLGGRKITRFYSSEKYGDHVSAALGAYNRQIDPDRLKVPVSGTRVRHDPYGQRRY
VDPVVYRDLITKVVFLGAPSTGKTTLASYLANLHGTQWMPEYGREYWAEHQIGRRLSKEQLLEIAEGHVGREEKLTEEAD
KYVFVDTDARTTLRFSMYYHGGALPKLHELAEAAAARYDLVFVCGTDIPYDDTWDRSGDAQRNLFQKQILADLNMRRIPY
FVLSGTLEERAARVNAVLAKYRKYRSLVEVLA
>lcl|NZ_CP026520.1_prot_WP_042234635.1_5234 [gene=pnuC] [locus_tag=PC41400_RS26435] [protein=nicotinamide riboside transporter PnuC] [protein_id=WP_042234635.1] [location=complement(5800427..5801125)] [gbkey=CDS]
MKKTGTMIVLLAAMLTIAWLTSSSAFEIAATTTGLLCVWLTARENIWSWPISLVNVACFFYMFWEVKLYADMALQVFFFL
LSIYGWIVWLTKREGAGVRPTRKIAPWLILPLVLFLIGATTGWGYVLHKYTDASIPYADAFIATLSIIAQLLLSAKILEN
WTVWIAVDVLSIGMYAYKGLDAIAFLYLIYLGIAIAGLITWKRNYAAAGQEGFSLTPEPAGNPHSTERGAWK
>lcl|NZ_CP026520.1_prot_WP_042234637.1_5235 [locus_tag=PC41400_RS26440] [protein=NUDIX domain-containing protein] [protein_id=WP_042234637.1] [location=complement(5801192..5801962)] [gbkey=CDS]
MNPNLAEENYSAGKYRTPDGAPADIVIFTITSTERGTVKKSLPHRELQVLLIQRKGWPFAGQWALPGGFCKETETVYDCA
KRELREETGVGEVHIEYFNVYSTPGRDPRGWIISHAFFALVHEQYLAGRQAADDAADVKLFSVEEALGMELAFDHKQILQ
DALDQVRRQMLTTIIAKEFLPEEFTISELYQVIRTLVPEFEEKNFIRKITSTQSRKGIIEEARDSRGERKVSNRYSQRAA
QLYRFTGDVPQLSLYS
>lcl|NZ_CP026520.1_prot_WP_042234640.1_5236 [locus_tag=PC41400_RS26445] [protein=class I SAM-dependent methyltransferase] [protein_id=WP_042234640.1] [location=complement(5802035..5804317)] [gbkey=CDS]
MLTSLKAIEEYRRGAEPPVPAYTWLRFLVSAEERIVNLERIRSIHELADANPVLDYVERTLRILDGLKLSFWVKEILETV
LVWSETAKGGTVRQRMQWQADGVNLFVHNIGSAQLFDRFNEMGETGTDKLLVIRTLIETHGLLGQYIRGEVPFEGNEPLT
EIVKRGLLTPEELQSLLIPLNHCIIGAVSPELWSTVRSEVEELTAHISSGEQPGVWSVRERLRKLRSGSIRQGENFEAEY
EKVAEALEATERVRVDKAGEGMPQGGQVFVGEALDGGTRQARQPARTKQAEHISQAGYSGLALERALQPLAGKTLWYVES
ALQEFSLEEVVKVFMLALQTGATTDSAMTMTDEPVHHISFERLMNAMYYDYKGVKKINVYKKRIIEKFLRELAFEHIGSS
GAPVNPHLAYKVERSPSAADTVFFTFEFSAAAEKLIEFCMEAEKSPLYEKAVLLLFDLFELRRDAYDRFHNEEEYLETMN
RTADYKSVMLEFITGEKVLDIGPGGGVMLDLIEQRLPDKQPVGIDISTNVIEALNRKKQLEGHRWEVLKGDALNLKDYVH
PGTVGTVIFSSILHELYSYIEWNGSRFNLDTVAAALRSAFDVLAAGGRIIIRDGIMTEPAEMKRRIRFLAHEDKEWLLRY
TGDFAGRQIQVEYAENGDAVMPVNDAMEFLYTYTWGEEAYVHEVQEQFGYFTPSGYADFIGKVLGPEAVIVEFRHFLQEG
YTDALAGKVVFMDEKGAPVPLPDSTCLIVIEKRGETHLDN
>lcl|NZ_CP026520.1_prot_WP_042234642.1_5237 [gene=pruA] [locus_tag=PC41400_RS26450] [protein=L-glutamate gamma-semialdehyde dehydrogenase] [protein_id=WP_042234642.1] [location=complement(5804547..5806118)] [gbkey=CDS]
MTTNQTTIAPFKNEPFTDFTQEDNKKAMQEALAKVKQELGKEYPLHIGADKVTTEDKIVSINPGNIDQVIGRVSKATQEL
AERAIQTALSTFETWKRVPSRERAEYLLKAAELMRQRKHEFSAWMILESGKNWAEADADTAEAIDFMEFYAREAIRLSET
NTHQPLTKLDGEDNRLTYIPLGVGVVIPPWNFPLAICAGMATAAVVSGNTVLVKPASTTPVIAHKFVALMEEVGLPAGVI
NYIPGSGAEVGDYLTTHPKTRFVSFTGSKEIGLRINRLAAETAEGQIWIKRVIAEMGGKDGIVVDETADLDAAAAAITAS
AFGFQGQKCSAGSRAVIVESVYDEIVEKVAELTKKLEIGLQDENYAAGPVIDENSYNKILEYIEIGKKEGRLVTGGGKAE
GNGYYIQPTVFADVDGKARVMQEEIFGPVLAIAKAKDWKEAIAMYNDTEFGLTGAFFSKDESRIETALQDMHCGNLYINR
KCTGALVGVHPFGGFNMSGTDSKAGGYDYLLLFTQAKLTSRKL
>lcl|NZ_CP026520.1_prot_WP_042234644.1_5238 [locus_tag=PC41400_RS26455] [protein=proline dehydrogenase] [protein_id=WP_042234644.1] [location=complement(5806152..5807066)] [gbkey=CDS]
MEAIMRNMFQALGKSHSANKMAKRFGLRFGAARFVAGESIQQAIEAVRGLNSDGRLATLDHLGEFVFSEEEANESAEMCI
NTLEAIAASGVNSNLSLKMTSLGLDISKDLCLQNMRRILDCAKKHGNFVRIDMEDYEHCQITLDILHELRDEYDNVGTVI
QAYLYRSEQDIRDLNGIHANLRLVKGAYKESPKVAFPEKKDVDDNYKKIIKQHLLNGNYAAIASHDEAVINEMKAFIRDM
KISGDQFEFQMLYGIREDLQKQLVAEGYRVRAYVPYGIDWFGYFMRRLAERPANVWFVLKNFFK
>lcl|NZ_CP026520.1_prot_WP_042234649.1_5239 [locus_tag=PC41400_RS26460] [protein=sigma-54-dependent Fis family transcriptional regulator] [protein_id=WP_042234649.1] [location=complement(5807277..5808989)] [gbkey=CDS]
MFGLEHLQLDTSFLALEGECPVSQAKALLEQVRFVIVDTAPGAAYIISYQEAKFLDAFHPDLTIDEWVRRTGWPCSAVLP
MSRLGDLQTDWQKPVLVLGEDSGKLGIVTAEAMMNCLQQEKRQLVSYLDTLVETVNDAVTAVDREGRVIYWNRTAEQVYG
IGKQDIIGRTIGEHFEKESIMLHQILDEGRTIRQVYHQPAPDTHVLISASPIVENNQIIGGIATEQDITRIVRLNEELYS
AIPLRIEQDKPFSSIIGMGHDIKRSLEVARKFARFSTPVLLVGEPGSGKEMLAQAIHYSSERKDAAFVSLHCGAIPGGLL
ETELFGYQGGAFTSGEDQGKEGKLEFADGGTLFLEEVGEMPLDIQLKLSQYLRDKSFHRAGGEESIRTDARIIATTHTSL
QPLMEQGLFLEDLFYQLNVMSIEIPPLRERREDIPELVQAFIREFTEHYHKPVPKVDADVLNVLMNHDWPGNIRELRNVM
ERAVILCEGDTITGVHLPPGLSSKMQHDEFFADRQETGAAKPHSAFQQEDHLIEEALRRAYGNKSAAAKLLGISRGTLYK
KMREYDIHTE
>lcl|NZ_CP026520.1_prot_WP_042234651.1_5240 [locus_tag=PC41400_RS26465] [protein=methyl-accepting chemotaxis protein] [protein_id=WP_042234651.1] [location=complement(5809137..5811140)] [gbkey=CDS]
MNQVWKKMKLNNSLKTKLIAAFLLLSLVPLVVTSVVLLNMAKTNTENEIAHNMKNLAKSNADSVDYWVRQRISMVEEMIK
KHPEMAKGEKTQILNMLKTVGEIDKEFEYFAYVEKDGTSTNTEGLVSKVGEREFFKKMVETKKTVVSDMIINSKTGKPVI
VIAVPIMQGSEFLGAIDAVINPEILQKLIEGVKVGESGFGYLISGEGTYMTHPDKERIGKKLDEVMQPSELKKFKEQVLA
NNDGEVDYKSEENIDRTASYYTVSATGWKMVVLTNDSEILGALQKFVRTTFLIILIVCILVLLISWQIGRVAVAPMLHLS
EVLKKVAGGDLTPRLKVTSKDEIGEIAQNTNKMLESMSTIIGDVSLMAEQVAASSEELTASSAESVEASKHVGQSIEEIV
RGSETQVQSAEQSSKAMEEMAGGIQRIAESSSTVADTTNRTAAEAKQGNNAIQMAVRQMDSIRDSVTATSGELKTLGEHS
RQIGDIVDAITQIASQTQMLSLNASIEAARAGEHGRGFAVVAGEVKKLAEQSAKSAAHITELIAHIRTSTDNAMTTMDRG
VAEVGKGAEIIHIAGIVFQDIYTAVQNVSDQIREVSAATEQMSAGSQEVSASMMEMLQISQGSSDSAQMISAASEQQLAS
MEEVSSASESLSRMAQSLQESLSKFKI
>lcl|NZ_CP026520.1_prot_WP_042234653.1_5241 [locus_tag=PC41400_RS26470] [protein=hypothetical protein] [protein_id=WP_042234653.1] [location=5811300..5812244] [gbkey=CDS]
MKWFTDYKEELELVFEEAVRRVAAFPAPLHTAGMDYLDRFHPLKKDSTKNYICYLLPFWLKEQTRLPADTYRTLSVANVF
VMLHFFVQDDFMDTDTGADERRLMLPLSNLFQAEYSRLYREYFPADSPFWAYADTYLRQWAECVAKENEADYFRQDPVRI
AYKAAPVKLASTGALLLSGQEELIEAFSRETDHTLLALQMADDWADWREDLDTGSYNCLLSLIRSRRGLKPEEPLSEDNV
KQAVFIHGCLRDYAEAAAACSTDARTGLFHAPHMDAFHQSLLNDLLAEASHIDKERGLLGLGGFTHLLSKRNKS
>lcl|NZ_CP026520.1_prot_WP_042234655.1_5242 [locus_tag=PC41400_RS26475] [protein=NHLP leader peptide family RiPP precursor] [protein_id=WP_042234655.1] [location=5812307..5812555] [gbkey=CDS]
MSLVPAEEILKNQIIEKAWADPEFKKALLNDPKSALKQAFDIELPESADVKVLDEAPGQYFLVIPPNPAEVLSAKSNVNS
VW
>lcl|NZ_CP026520.1_prot_WP_042234658.1_5243 [locus_tag=PC41400_RS26480] [protein=HAMP domain-containing sensor histidine kinase] [protein_id=WP_042234658.1] [location=complement(5812571..5814475)] [gbkey=CDS]
MTKSRVTEFLASVFTGIVLLTLLFVPATARGQSPLDTIKITEWAYKWDSQGTSFDGTQAPVGADGWIPANEKGDMPVKPE
GVSALWVKIELPRLDWDEPGMLIGKLYGHRIEIFTDERKIYKAGHTYVYDKNRLLIPLSAKESGKTVYIHIETDSDRIGL
QKNVNIGDYQALLPGYVKSNLTDILLGFSFVFIALLMLICTVFLNRVQLAKWISLSLVILAIGMLMIGSSQFTYTFFEDQ
GKWIESLFDLALLVFLPTLTVYFEQILGHGAWSLLTKYRWFQIGFSVLCVLFQLVNHLASYRYDKAYYFMTVTMLGLTML
IQFVLLIGMAATHALKGSKDGVTLTTGIGLFAASGIVDLLVFYAKNEQYEFIYWKFGIVCFVTSLIIILGRGFARDHEKI
VKYSKELVMYNNELQRAEKMEIISDLAASVAHEVRNPLQVTRGFLQLAAEKLDQKEKGYMNMAIEELDRASNIITDFLTF
AKPQLEHITSLNICQELKQIEGILVPLANLQGGEITLEVPEDLYIEGNSSKFKQAFINIIKNSIEALDGQGQIQIWAYKQ
KTEVWIHIRDNGEGMETKELSKLGEPYFSTKTKGTGLGLMVTFRIIEVMHGRLEFKSEKGVGTEAIVRFPSIPG
>lcl|NZ_CP026520.1_prot_WP_042234660.1_5244 [locus_tag=PC41400_RS26485] [protein=nucleoside triphosphate pyrophosphohydrolase] [protein_id=WP_042234660.1] [location=complement(5814630..5814962)] [gbkey=CDS]
MTIYNKLVRDKIPEIIRENGQNCRTRLLGEEEYKLELRRKLVEELNEYMEARDDVQAVEELADLLELLHALTDLHGSSPE
ELESVRERKADKRGGFRDRIYLLETEDAGK
>lcl|NZ_CP026520.1_prot_WP_042234662.1_5245 [locus_tag=PC41400_RS26490] [protein=ribonuclease Z] [protein_id=WP_042234662.1] [location=5815183..5815932] [gbkey=CDS]
MDVLILGSASASGSRERDNTYLLLQHDEGSWMIDVGGNPLGKLKQAEIPLHSIKGVILTHFHIDHIFGLPSLLWGMWLGK
RKEPLIICCPQDHQDQLGDILRAYRTDEWPIGFEIKVQPFDCAEPGTLFVQGDMTVSVFTSLHAGSTSGIRVACGDKVLV
YSADTAPSEWIRSQPRIDVLIHEATQARGDMSIHTSLEGILTFYPVDRIPQVIAIHLTDDEPYDEVLAQYKKTGTPGVLL
AHDMMRVSL
>lcl|NZ_CP026520.1_prot_WP_042234664.1_5246 [locus_tag=PC41400_RS26495] [protein=3'-5' exonuclease] [protein_id=WP_042234664.1] [location=5815946..5816614] [gbkey=CDS]
MKVIVYDLEMTVTRKKIHVSEIIEIGAAAVEFTDGKPVVTGTFQSFVKPDRSPVISASTQALTGIKQSELQSAEPWPLVL
ERLLEWIGDGDYYLCAWGPDDLIQLGSDCRMHGVKTDWICNHNNLQKLMAGLVQQEKQQQMGLKTALEYWNIPFDGSHHR
ALDDAMNTAKLFIHLHDRLVLERNQAGDLPLYTTEVVYKTGDSFTNNPFEKLALFAQKEPTS
>lcl|NZ_CP026520.1_prot_WP_042234666.1_5247 [locus_tag=PC41400_RS26500] [protein=glutathione peroxidase] [protein_id=WP_042234666.1] [location=complement(5816672..5817217)] [gbkey=CDS]
MSVYPYSARNIRGAETKLEDYKGKVLVIVNTASQCGFTSQYADLQKLYEKYRDQGLVVLGFPCNQFGGQEPGTNEEVNTF
CQLNYGVTFPLFEKIDVRGEEKHPLFAYLTEQAPFEGFDLNHPSGKMLNAFLSEKQPHLLQGNDVKWNFTKFLIDREGSV
VKRFESGVDPLDMQSAVEAVL
>lcl|NZ_CP026520.1_prot_WP_042234668.1_5248 [locus_tag=PC41400_RS26505] [protein=peptidylprolyl isomerase] [protein_id=WP_042234668.1] [location=5817470..5818168] [gbkey=CDS]
MTGKRKSFTGLLLLMMALLVIAAGCGKAKTDGSGTATPAPTPSPTATGSAGGSEPPKTDGKNPVVTIEMDNGKIIKAELY
PEVAPNTVNNFISLIKKGFYDGKIFHRVIPGFMIQGGDPDGTGAGGPGYGIKGEFTSNGFKNELKHTRGVLSMARAQDPD
SAGSQFFVMVADSPHLDGQYASFGKVTEGMEAVDEIVNQPTEANDRPKTPPAMKKVTVDTLGVDYPEPKTIK
>lcl|NZ_CP026520.1_prot_WP_042234669.1_5249 [gene=ybaK] [locus_tag=PC41400_RS26510] [protein=Cys-tRNA(Pro) deacylase] [protein_id=WP_042234669.1] [location=complement(5818336..5818815)] [gbkey=CDS]
MGTQAKTNACRMLDKLGIAYTLHEYEWDEDQLDAGTVAGKVGIGIDSMYKTLVLRGDKTGVLLACIPGGKELHLKALASV
SGNKKVDMVPVKELQALTGYIRGGVSPLGMKKNYPLYLERTALATEPVSISAGRRGLQIFLGGKDLAAAADAIPADITH
>lcl|NZ_CP026520.1_prot_WP_042234671.1_5250 [locus_tag=PC41400_RS26515] [protein=AraC family transcriptional regulator] [protein_id=WP_042234671.1] [location=complement(5818848..5820035)] [gbkey=CDS]
MEEKELGEKPVAREIQAVAEKQASRETSENREQEREDREARKGLKEAQEAQEAQEAQEVQEVQEAQEAQEAQEAQEARAL
EFQEVPASGETREARLNRMQRTHLEQVIRIGSIITMYYFEHGKDYVFRGERHNFWEFLYVDKGEIEVVTDDAGHVLKQGM
AIFHEPNEFHSFRAAPGRAPNVIVLTFDCDSPAMNAFARGVLRLEDEERDLLARIIREGENAFEFPFEYPLKRRERALPG
SEQLLRCYLEAFLIRLLRQAPFDENASSAQLSSPARQKNQEELAAKVIAFMEANVESPVTLDEISRHFHIGKTLLKESFK
AQTGTTIMKYWGRIKIGQAKLLIREETCNFTEIAGRLGFGSVHHFSKTFKKASGMSPSEYARSVKARMGTGDSRA
>lcl|NZ_CP026520.1_prot_WP_042234673.1_5251 [locus_tag=PC41400_RS26520] [protein=Gfo/Idh/MocA family oxidoreductase] [protein_id=WP_042234673.1] [location=5820179..5821333] [gbkey=CDS]
MNLGIIGYGDRISHVASEIMKQDPDCKITAITDIRLGELREQFREKGREDISFYTSADEMLGNEPLDGVLIGTRCSLHTP
MALKVLSRHLPLYLEKPVATTMEDLLRLKQGYEASRSPVVVSFPLRVTSIVDFVREIIDSGKIGTVEHVQAVNNVPYGAV
YYQNWYRDEAETGGLFLQKATHDFDYINSVLRIAPTDVAAMTSKQIFKGTKPAGLKCADCEEKRTCPDSTAFPSKDIPQT
WEYCLYAEDTGNEDSGSALIRYESGMHVSYSQNFFARRGAASRGARFLGYKGTVEFDFYTNRVQVFMHHTPRVETYEIGA
GDNHFGGDSALAANFAGLMRGAASSVATMDDGLLSALMCVQARESAASGTFRPIRYDTPAVYTP
>lcl|NZ_CP026520.1_prot_WP_042234674.1_5252 [locus_tag=PC41400_RS26525] [protein=GNAT family N-acetyltransferase] [protein_id=WP_042234674.1] [location=complement(5821382..5821843)] [gbkey=CDS]
MTTDSVSIRPATAEDTRQLHDLMIQYIVDFYKRPRPDERELDKLIGHLRENPSAGLQFVAEKDGTLVGFATLYFSFSTLQ
VKRTAVLNDLFVDASARGHKVGERLFRSCLAYIREQGFAYMTWETAKDNTAAQSLYNKMGGHVSTDWLVYEIE
>lcl|NZ_CP026520.1_prot_WP_042234675.1_5253 [locus_tag=PC41400_RS26530] [protein=alpha/beta fold hydrolase] [protein_id=WP_042234675.1] [location=5822046..5823008] [gbkey=CDS]
MNAVEKRIHTLHQYKPALTLDPSAADHFWDKKLDLFKNKPLNARVEQRDSLMNGVDTYHVTYEGFDETPIHGWFLVPSFM
QKKSYPCLVMFPGYTADKGLPERYAAWLLQGFAVFAVDVRGQGGETGNRLDYDAGTSTGWMTQNILDKDRCYYMAVTLDA
YKAVEWAAGRPEVRASKIAVVGGSQGGGLSLITGALHEKVSLIVADIPNMCHMDFGVLNSVGSLKEAADFIKRNPDRLEP
VFETLAHFDMMNLAHRIRVPVRMTVGLKDMVCWPEAIFAAYNRIEGDKHIDVHPFSGHEVSERQQQEHRRYLAEWAREAQ
>lcl|NZ_CP026520.1_prot_WP_129112539.1_5254 [locus_tag=PC41400_RS26535] [protein=hypothetical protein] [protein_id=WP_129112539.1] [location=5823114..5823311] [gbkey=CDS]
MGRGSRAGQTQPESDPAKRLRHAACPLQGLAPGLSARSSLRHCPLSLRHPLESPQQPSSIWSVFH
>lcl|NZ_CP026520.1_prot_WP_042234677.1_5255 [locus_tag=PC41400_RS26540] [protein=GntR family transcriptional regulator] [protein_id=WP_042234677.1] [location=5823376..5824083] [gbkey=CDS]
MASTRQIPFITSGKSLGGQAYEALRDSIITLKLEPGQTIFESEIADTFRISRTPIRDAFQLLVMEHLIEVLPQRHKKIAR
ISESKVKESSFVRLSLESSAFRLAAGSWDSSEKFVQAERQISRILREQSEAAEHQDVVQFLQLDEAFHKCVLELTGNATL
LEVVYHMRGHLNRLRYLAMKELGLTPKLVEEHLELFDSLKKRDEASTVQLLEQHLGSLSYEIPQLREQFPDYFID
>lcl|NZ_CP026520.1_prot_WP_042234679.1_5256 [gene=uxuA] [locus_tag=PC41400_RS26545] [protein=mannonate dehydratase] [protein_id=WP_042234679.1] [location=5824236..5825273] [gbkey=CDS]
MKMTFRWFGENDPVTLTNIRQIPGVTGIVSAIYDVPVGEAWPLDKITRLKQTIEDHGLRLDVIESVPIHEDIKLGLPSRQ
AYIDNYKTTLRNLAKAGIDTVCYNFMPIFDWTRTSLNYELEDGSNALIYEEEVAQKMNPLTGELKLPGWDTSYTDDQLKT
LFDHYKKVDEEKLWTNLAYFIREIIPVADEEGIKMAIHPDDPPWPIFGLPRIITNRENLDRFLGLYDSPNHGLCMCSGSL
GAHKDNDFPAMLRHFGHQKRIHFVHARNIKRIGDKSFCETSHLSEHGSVDMVEVMKALKEVGFEGPLRPDHGRMIWGETG
KPGYGLYDRALGIVYLNGIREALNH
>lcl|NZ_CP026520.1_prot_WP_042234681.1_5257 [locus_tag=PC41400_RS26550] [protein=SDR family NAD(P)-dependent oxidoreductase] [protein_id=WP_042234681.1] [location=5825387..5826178] [gbkey=CDS]
MPDKRLEGKVAVVSGGGSGIGKASAIRLAQQGAKVVMLDRTPENAEKTKKDIESAGGEALVIECDISKPEMIEQGLGQAV
ARFGGIDIVFANAGINGTKAPIETLEIDEWEQTININLRGTFATVKYAIPHLKEKGGSVIITSSINGNRVFSNIGFSTYA
TTKAGQVAFMKMAALELARYKIRVNAICPGAIDTNIDDNTYPTDDLKEVQIPVEFPEGDHPLKGAPGKPEQVADLVLFLA
SDESSHVTGTEIYVDGAESLLRG
>lcl|NZ_CP026520.1_prot_WP_042234683.1_5258 [locus_tag=PC41400_RS26555] [protein=STM3941 family protein] [protein_id=WP_042234683.1] [location=complement(5826300..5826839)] [gbkey=CDS]
MQPVHQPNDLIIYPSTKKTIGLVLAALVFVVLGIVFIGAGQSEEERSLALVAIGGVSVVFFGACLVYLVYRLFNRKPSLI
VNDEGVVDHSSAIGGGELKWADIEEMMLYDYMGQRFIGIRLHDAETYLARQSGFKKALMKANQGLVQATVNITQTAVKMP
LEELYALMMEKWERDRAGA
>lcl|NZ_CP026520.1_prot_WP_042234686.1_5259 [locus_tag=PC41400_RS26560] [protein=tetratricopeptide repeat protein] [protein_id=WP_042234686.1] [location=complement(5826913..5828889)] [gbkey=CDS]
MKLFNGYYIKKGRQYIKEGNFAKALQYYEKVDYGKLDTAEKIRVAEVYHECGRTEHALSLLGDIIGRVSSDQAHERRAHI
LQEIGREEEAMRDLDEAIRLNPEPYMYWYTRGIWRKDFQDYEAAAADLEECVKREPEDSVVSSYYELGMCYWRMEDYGRA
EQTFRKVLSYPDRDIPIFHYRLGKTLENLGRLEEAAASMRKAVDQVIEWEYISDKGQRLVLTRTGYGAGAFRIFMEEAER
TCGFRKDLAGLYAGLKRYGEALEEINRALETYPDNDGLYLQRVAVYRSSGRWEDAERELDVLEKERDVEERVLWMERVLL
YRAQNRDSEAVPLLHKLYEQDSGNPIVCYWLADAYLDLGQTGKALEFNKRLLELEDNDPLNEQQRGLILEETGAYGQAAS
AYGRAIAMQDRADFRMKRSYAYFMDSRFDEALMDLHHAAELEPGISEGAGYHHAMGRVLEEMGHLDMAIASFTKAIDIQP
EHAAYYESRAKCYLTLRELEKAELDCIVGYDKDASYTDLISLRGNVLYLKKEYAESRNLASQFSRLHPDSPSAFFNLGLV
CFKLEGEEDAARNAFDQALALDPSYGACYIYKAYLDYGRLDFREAIENLANWSLFTFPELSLEERLEHLEELDGLNETVL
EGAARRITELYQGHVYLS
>lcl|NZ_CP026520.1_prot_WP_129112540.1_5260 [locus_tag=PC41400_RS26565] [protein=chemotaxis protein CheA] [protein_id=WP_129112540.1] [location=complement(5829205..5831379)] [gbkey=CDS]
MLSEYKEIFLEELEEQLQVMDGEILKLEQSGESGIVIQSLFRAAHTIKGSSAAMGFEEMKQLTHEMEHLLDRVRSGQLAV
SGALIDLLFTALDILKQLKHDIVGGESSSADISDCVRALRNFGEDQDLDGDRPDAEEMPAHRGKPSLSLDLRLRVQERME
LGQKLFWVDIRIASECLIRGARAYVIHSALNEYGDVLLADPEPEGLNEEGPGDLLFLYAGNQSREELQAFADGLMEVERA
VVEPLAQEEAAVSGAGAAERAAGSAAGAGESAAPDAAGTAGPAWSAPDATGTAGPAGSAAAAAADAPPHAAASASRPSAE
DASAPAKAKSPTIRVSVERLEHLMNLVGELVIDQTRIHQVERTQRRRFADETVDELGHIADHLSRIIGDLQESVMKTRML
PIEQLFNRFPRMIRDLSRMLGKEIELVLEGKDTELDRTLIEEIADPLIHLIRNAVDHGIEKPEVRTLSGKTAGGVLTIRA
AHEDNQVVIYVQDDGAGIDPARMIRSALQKGVITQEEAGQLSEREAVELIFRPGFSTASTVSDVSGRGVGMDIVRSHIEK
LNGLIDIETRLGQGTCFKIKLPLTLAIIVGLQVKLGGRTFIVPMSNIAEIVRVNPDEIRSIQGQSVIVLRNQVIPVARLH
DYFQIPYTANTGSHIPLVIVGSAEKRLALAVDELIGNQEIVIKSLGPYIGKVDGIAGATILGDGNVALILEIASIIGRTG
GRPH
>lcl|NZ_CP026520.1_prot_WP_241688740.1_5261 [locus_tag=PC41400_RS26570] [protein=chemotaxis protein CheW] [protein_id=WP_241688740.1] [location=complement(5831392..5831844)] [gbkey=CDS]
MNDREDLFMQGSGQEQYVAFGINREQYAIRIYDIHEIIKMQAVTAIPNVKPYVKGVINLRGRIVPVISLRTLFGMEEAEP
TKASRIVVVHHLEDTVGIMVDYVSKVTTFTDIQPPPERVGGVQGGFFTGIGLADSGLAGILKLDEVLLHE
>lcl|NZ_CP026520.1_prot_WP_042235444.1_5262 [locus_tag=PC41400_RS26575] [protein=methyl-accepting chemotaxis protein] [protein_id=WP_042235444.1] [location=complement(5831909..5833513)] [gbkey=CDS]
MKWYRNLKTSAKLVSAFVFIAILLVFTGFFGLFNMSKLNNSLSSMYENNLVPLRVVSDASFYYQLIRTDLRDMNDLAETP
AQKDEYKKKIDENVKKISEKIAVYEKTDITQKEKDLLQSFHSVWESYQKQLDEAVKVDYGDDREAFKALLASGSFKTKGD
EVMKFFDDMFEINVKLAEERNAEGAALYKSSSNVTFGVMIAAFLISIGLGYAIARIISRPLNQVVGLVGKVSAGDLTEVT
DIDTRDEIGQLAKSVNEMVYSLRSTVGGILASAESVSAAAEQISASTQEIASGSVSQAGAAQTIHELFKELSTAITSVAT
SAEQASGVSSDALNIAQQGGKIVNSSIESMNQVNEQVSRLEEDSNKIGEIIEVIDDIAEQTNLLALNAAIEAARAGDQGR
GFAVVADEVRKLAERSGEATKQITAIIKGMQENTRQSVKAVGDGVVLSRQTGESFHHIIGMVNESANKVMEIAAASEQQA
AQSTEVMSSIESISAVTEQSAASSEETASTAQSLAGLAEELNRTVAVFKISKSA
>lcl|NZ_CP026520.1_prot_WP_164977157.1_5263 [locus_tag=PC41400_RS30000] [protein=hypothetical protein] [protein_id=WP_164977157.1] [location=complement(5833510..5833686)] [gbkey=CDS]
MEKTRAIRALWDNSRRRTDYNPERNEGKLTSEEIDDILNDLEAELKVIEGRKGRRYGR
>lcl|NZ_CP026520.1_prot_WP_241688741.1_5264 [locus_tag=PC41400_RS26580] [protein=chemotaxis protein CheX] [protein_id=WP_241688741.1] [location=complement(5833728..5834201)] [gbkey=CDS]
MIEVQNPNGIQDLLNSTIASMSQVFPIPIDTKPPSLVQNAVIQGEIGVLIGIVGDVEGRLVIEGDMQTFGKLGQSMFGMP
LEGEMLHSFVGEMANMIAGNTSAIVYQKGHKIDITPPTVMVGQMQLYGFEKGISIAVSLQEVGTIHMILLLQKTEAA
>lcl|NZ_CP026520.1_prot_WP_042235439.1_5265 [locus_tag=PC41400_RS26585] [protein=response regulator transcription factor] [protein_id=WP_042235439.1] [location=complement(5834538..5835206)] [gbkey=CDS]
MIEILLVDDHPSVMEGTKMLLEQDTNMKVSLAYTAKEAMELAVARRFDVMLIDLHMPTMNGIDLAKQILNLIPDAVILIY
TGFEFNNHFNLMIEAGISGFVLKTANKEQLVTAVRCALRGEAVLPLSLVKHLRRTTGSQFAESASGNPETQAVSSKEYEI
LKEIARGRSNKEIAASVLMSQRSLEYCLTNLFHKLNVKSRIEAAMKAKQLGLLSDEDFPKMA
>lcl|NZ_CP026520.1_prot_WP_042235437.1_5266 [locus_tag=PC41400_RS26590] [protein=PAS domain S-box protein] [protein_id=WP_042235437.1] [location=complement(5835199..5838738)] [gbkey=CDS]
MNEFIIGAVVCILLLTVQILLILYKRNLEKWKHHEELVKKYGLLLKIMHTVDVGTWSYDYSSRRVPFTSEALLKITGYRP
EYFATQQAWERIIHPDDFQRYKVACANVRQGISDYSEYRIVHATGEIRWVQVRMIPTLDETGGRFVRLDGVVLDITARKA
AEEALLRSEQRYRSLFEHHSDVIVELDLHGKVLDINSAALEITGNGRNIANPLSLIELFGLDSLKRISVYFEQVVQGTTQ
TFEMTSYHKEDRMYHWDIKMVPVYVNGAVTGIFVLCRDITSQKRTEQSLKESEALYRLIAENMTDLVAVIGTDGKFRYTS
PSYKARLGLDGATEAGLSLCDIMHPEDEKILRLKLEEMPESACVWTARCRLIHRNGSELHFDCVANPVQGAGGEVESVIV
VSRDITERVRIERELVESEERYRHLIELLPQPLTMIRDERFIYTNPAGLELFGAVQPCEVIGRSIYEFMHPDYVEQARGR
AVAMAQKKYGPALEYKMVRLDGGVIDVEVTGIYDGLTNSSLHVHTDITARKEMERALRDSEERYRSLVEMSPVSIALYKD
GQVIYANPAAVRVLGAECPEEIIGTTPQDWIQPGSPVVREDVIREFGVNDPIDLAITRRDGKVVDMLATAIYDARSEAFL
VFFEDITVRKQAETALEESERLSRNLVELSPIPTILTRDHKFEYVNPAGLDLFGAGTSCSLLGRPIYDTVHPDYRDKVRA
RLHTVYESKEATRITENKMIRLDGRIIDAEMVSIPIPYIGRDVAMTVIRDISEQKKAEMDREYAEQIVRESKDRYYMLQT
SLDRFSSDLFGLMNVQELDRRLIQEVREMIRTEKISLVEVDKQNKITVKNGSPDIPQHVWEEIVEGCSDPLPVCEIIDTP
DGHLIRIGEIKGKSYLLCIGEKPPSLGFQSKRVWIKTIARYASVLYDNFRVIEDLTQELRELATRQKTPSWLIRLLFNLS
ENESKRLSQDLHDGALQEQIIWYRKLDEISSDDEVPPVLREKLAQITQGLLDVIYQIRITCNELRPPMLKEEGLVASLEA
LFEFTQLRSDYSIAFDYTAFHGSLHDDDLLIGLYRIVQELLANATKHSNAAKVQFKLSAHSDRIHLSYEDNGIGMDVSGM
VESFNSMGVYGIKERVRSMDGDVEFYSSPNNGLAVFINVPVKQNQKFEIRNWSVRDNDD
>lcl|NZ_CP026520.1_prot_WP_042235435.1_5267 [locus_tag=PC41400_RS26595] [protein=futalosine hydrolase] [protein_id=WP_042235435.1] [location=5839002..5839703] [gbkey=CDS]
MSPDFYSASSPLPAGGTPHIKRVLIMTPVEAERDAVLRGLAGAPGFRVELAGVGPVAAAVSTTAALAAAEYDLVITAGIA
GGFEGRAAVGSLVVSSEILAADLGAETPGGFLSLDELGFGSARVPVDAALAGRVAEALAAAGLSVSTGPVLTVSTVTGTA
EHALKLAGRVPGAAAEGMEGFGAASAALARGIPCLELRSVSNAVGPRNKAAWRIPEALAMLEKASSVLPEVLL
>lcl|NZ_CP026520.1_prot_WP_042235433.1_5268 [locus_tag=PC41400_RS26600] [protein=1,4-dihydroxy-6-naphthoate synthase] [protein_id=WP_042235433.1] [location=5839700..5840530] [gbkey=CDS]
MKIAFSPCPNDTFVFHAWVHGLVPGAPELDVLYADIDITNGLAARPGGPEVMKISYAALPWVLSDYALLPCGGALGRGCG
PLVLSRDSADPASLSGKRVAVPSERSTAYLLFRLWAAQQVPGGVGEIVVMPFHEIMPAVRDGKIDAGLVIHEARFTYPSY
GLSLLADLGSWWESDTGLPIPLGAIIAKRSMDLQAVAGWIRSSVEYAWAHPEASQAYVLQHAQEMSPEVAKAHIDLYVNS
FTADLGEDGYGAIEALLGRAAQEGLVPEFDLKMLRG
>lcl|NZ_CP026520.1_prot_WP_042235431.1_5269 [locus_tag=PC41400_RS26605] [protein=NADH-dependent flavin oxidoreductase] [protein_id=WP_042235431.1] [location=complement(5840954..5842072)] [gbkey=CDS]
MNSKFKPLFESFSLSSGVELDNRIVLAPMTNFSSHEDGTVSDAEIAYYERRSNGAGLVITACTYVTPNGKGFPGEFAGDR
DEMIPSLTRLASAIKGQGAKAILQIFHGGRSCPPELVPNGEIVSASAVAEEKEGSVVPRALTDAEIEGIIRDFGETTRRA
IEAGYDGVEIHGANGYLIHQFFSPHSNRRDDRWGGDVNKRMTFPLAIVDEVKRIVAEHAKQPFAVGYRFSPEEAETPGIS
MDDSIQLVNTLADKGLDYIHVSLMDFWSASRTMDDNRPRIEIILETVGGRTPVMGVGSIHTADDALKALETGVPLIALGR
ELIVEPDWVGKVADGHESEIKTTLTKDDQERLVIPDPLWNAIINTPGWFPVV
>lcl|NZ_CP026520.1_prot_WP_042235428.1_5270 [locus_tag=PC41400_RS26610] [protein=DsbA family oxidoreductase] [protein_id=WP_042235428.1] [location=5842469..5843191] [gbkey=CDS]
MKIEIWSDFACPFCYIGKRRFEQALEQIPFKNELEVVYRSFELDPNSPRDIGHSVHDMLSKKYGMSVEQAKTMNANVVAQ
AESVGLTYRMDTIVLTNTFDAHRLAHFANAHGKMHEMTERLLKAYFTDSLHLGDRAVLASLAEEVGLDKAAAEQMLAGDQ
YASEVRADEQEGAELGITGVPFFVIDRKYGISGAQPLEVFVQALEKAYGESKPLTILGDEGGAADGACADGVCTPPARDK
>lcl|NZ_CP026520.1_prot_WP_042235427.1_5271 [gene=speD] [locus_tag=PC41400_RS26615] [protein=adenosylmethionine decarboxylase] [protein_id=WP_042235427.1] [location=complement(5843296..5844138)] [gbkey=CDS]
MALTPEQRITLHGFNNLTKSLSFNMYDICYTRTREERQAYLEYIDEQYNAERLTKILNTVSDIIGAHVLNIAKQDYEPQG
SSVTMLVSEGPVDTAPTESFEESPGPLPEAVVAHLDKSHITVHTYPEYHPDEGISTFRADIDVSTCGEISPLKALNYMIH
SFDTDIFTMDYRVRGFTRDINGHKLFIDHEISSIQNYIPQSIKDKYHMIDVNVYQENIFHTKCKLREFDLNNYLFGYTKD
KLTAGEQEEITESLTLEMDEIFYGKNIYAATLEDVEGGDK
>lcl|NZ_CP026520.1_prot_WP_042235425.1_5272 [locus_tag=PC41400_RS26620] [protein=site-2 protease family protein] [protein_id=WP_042235425.1] [location=complement(5844301..5845359)] [gbkey=CDS]
MKGKVKFLIGCLVVLAAIAVASPGMALNIVVFLLSFAAAGGAALFVHELGHVLGAFITGRKVIKFSLGPITLSIPQKKII
FSWKNFYYVGNVLVDVADFKDEASYERNNKKQSVIFILGPIMSLVTGILALTLLKNSFDHLVVTLFGILSLAMGLGTLIF
SDGRLAKTISDPYESLYYYWNILFTIPHVKEESLKFLLLKSETYLFDKYIGNKKGITEKFTDLFVLYYSKYFSQVIGRER
IHAIDFMPFLMDALRDESGSRHNKMIVSHILCEEAMGWSLAGRQDEAESLYAFVQEHGVTEPITRLKVEAVLKRSVELGR
EYAAQRKALVQNNVFDYYEEKWLKERGLPGFK
>lcl|NZ_CP026520.1_prot_WP_042235423.1_5273 [locus_tag=PC41400_RS26625] [protein=GNAT family protein] [protein_id=WP_042235423.1] [location=complement(5845518..5846096)] [gbkey=CDS]
MNIQTVFAQLPVLESSRLVLKKIGEDNLDEVFDIYDNEEVFRHCGIIPKHNKETVRSMIGHFERDFGKKSRIKWGIFRKE
DGRMTGIAEAFDFNQKVDMVTIGYFLAEDCWGMGIATEAVRLLTAYLFTEAEVNRIQAEVMPANDVSKKVLIKNGFRKEG
TLRQAALWSGKGIVDLEIYAILKEDYDAAPDK
>lcl|NZ_CP026520.1_prot_WP_042235421.1_5274 [locus_tag=PC41400_RS26630] [protein=AraC family transcriptional regulator] [protein_id=WP_042235421.1] [location=complement(5846108..5847040)] [gbkey=CDS]
MKHEYADRIGKVIAYIEENSSRKLDLDALAGVSNFSKYHFSRIFAAVTGVTPAAYVTRKRLENAVRLLKDTDRTVLDIAH
SCGFESLSTFNGVFKRRHGKSPSELRRELQDRNFSLGDRKKREEPSSSYEYDESAGNVNHFLRRIWSMNVTLKELSDLEV
AFVRHTGSYLETGRAWETLGSWAGRNGLTPERQQFIGISLDDPSLVDEFTCRYDACVTLPPDFEREGHTGVSFKKLEGGS
YGRYAFYDTVDKLALAYANLYGNWLPGSGYEADDRYALEFCMNDPLRDPEGKAKVDLYIPVRPKDELIPL
>lcl|NZ_CP026520.1_prot_WP_156972873.1_5275 [locus_tag=PC41400_RS30465] [protein=hypothetical protein] [protein_id=WP_156972873.1] [location=complement(5847037..5847183)] [gbkey=CDS]
MNRPAKIAVAGSINMDLVVTAERLPIPGETVRSEPAVARFRPKGRRIP
>lcl|NZ_CP026520.1_prot_WP_042235419.1_5276 [locus_tag=PC41400_RS26635] [protein=DJ-1/PfpI family protein] [protein_id=WP_042235419.1] [location=5847409..5847966] [gbkey=CDS]
MSKKILIITGDAVEALEVYYPYYRVLEEGYEAVLASPQQKVLKTVIHDFEGWDTYTEKPGYLLESHVSFGEVNPAEFDAL
IIPGGRAPEYIRLDPNIPRIVGHFFEEGKPVGAICHAALVLAALKNNKYFEGRTLTAYTACRFDVESLGANYTEETLHVD
GNLVSGHAWPDLPGFMREFLKLVKS
>lcl|NZ_CP026520.1_prot_WP_042235416.1_5277 [gene=opuFB] [locus_tag=PC41400_RS26640] [protein=osmoprotectant update ABC transporter permease/substrate-binding subunit OpuFB] [protein_id=WP_042235416.1] [location=complement(5848107..5849627)] [gbkey=CDS]
MNTLAEIFRERQSQLLGAVADHILLSVIALFFAVLLSVPLGIYLTRKTRGAEGIIGIAAVLQTIPSLALLGMLIPLFGIG
RVPAVIALVIYALLPILRNTYTGIKEVDASLIEASRAMGMNSRRRLFKVELPLALPVILAGIRTATVLIISTATLAALIG
AGGLGDLILLGIDRNDSALIILGAIPAALLAILFDLLLRAFERMSFRRFVPAAGAAVLVVLLLLLIPYFTGGKDKNLVIA
GKLGSEPEILIQMYKQLIEQDTDLKVELKPGLGKTSFVFNALKAGEIDIYPEFTGTAISEFLKETPVSTDRKQVYEQAKA
GMLSKFGMEMLTPMAYNNTYALAVPTQLAEEQGLKTISDLQNVRQTIKPGFTLEFSDREDGYRGIQKLYGLQFADVKKME
PKIRYQAVQSGDINLVDAYSTDSELKQYNLTMLEDDKHLFPPYQGAPVLLTKTLEEHPELRGVLNKLGGKIGDDEMREMN
YKVNVQGVSAEQTAKDYLMGAGLLGK
>lcl|NZ_CP026520.1_prot_WP_042235414.1_5278 [locus_tag=PC41400_RS26645] [protein=ABC transporter ATP-binding protein] [protein_id=WP_042235414.1] [location=complement(5849624..5850586)] [gbkey=CDS]
MILFENVSKRYADDTVAVNDLNLEINEGEFFVFIGPSGCGKTTTLKMINRLIPLSGGTLWINGRKVSEYDVHELRWNIGY
VLQQIALFPHMTIEENIAIVPEMRKWDRAKIRARVDELLDMVGLEPEIYRSRKPKELSGGQQQRVGVVRALAADPQIVLM
DEPFSALDPLSREKLQNELLELQQRIKKTIVFVTHDMQEAMKLGDRICLMNEGEIVQLDTPQGFLQTPANAFVEQFVGGQ
KPLFGSRLTLEDVLLEIGPSDFPTHPAAIAKAEVPLEAALQLLADEEVLAVEKAGVIVGSVSRQALLKFMAGRLKEGART
>lcl|NZ_CP026520.1_prot_WP_042235412.1_5279 [locus_tag=PC41400_RS26650] [protein=DUF523 domain-containing protein] [protein_id=WP_042235412.1] [location=complement(5850939..5851415)] [gbkey=CDS]
MILVSSCLAGLECRYNATHSLVEKISELVGEGKATMVCPELLGGFTTPREPAEIIGGTGEDVLAGGAKVVDRSGNDVTGL
YASGAFKTLELARKVGATVVVLKENSPSCGSRAIYNGAFNGTKISGEGVTAALLRKEGFKVISEDELLEHGLSEYVSV
>lcl|NZ_CP026520.1_prot_WP_042235410.1_5280 [locus_tag=PC41400_RS26655] [protein=HAMP domain-containing sensor histidine kinase] [protein_id=WP_042235410.1] [location=complement(5851522..5852850)] [gbkey=CDS]
MKIRAKLMLSMGILIAFILFSLFAVTHIQFYIIRDLEYMREKPAFTALKENFQRHYAAGGNSWAGVRAEEMDSGSFFPEI
ALVSGGSLVVHKGQLPLRTLQDEGFPLFLTDAEGKIGTLFVMDAKQYKTYEFKSMWYGIFPVVVRVSLLFTFVAALVISF
FLSRRFTSPIRKLLQGIDSIKKGSGGDVKLPVARKDEFGAIARALQEMNDSLASLERSRKQMLSDVAHELKTPLMIMQGE
LELAQEIDEPLKQEKVSSLLDEVLRLNRLVHDVLELSRMEAGRAELRREPENIGALLQGLIEKTRYIAEDKNIRVRLRTP
EEDIYAPVEKHRILQALYNILINALHYTPAGGTVKIGAGTAVREGRLVVLLTVEDNGSGIASEDLPHIFNRFYRADHSRT
RPSGGTGLGLAIAEQNILLHGGSIGVQSESGKGTVFTVLLPV
>lcl|NZ_CP026520.1_prot_WP_042235453.1_5281 [locus_tag=PC41400_RS26660] [protein=response regulator transcription factor] [protein_id=WP_042235453.1] [location=complement(5852840..5853538)] [gbkey=CDS]
MTVKLLIVEDEEKILALMETFLRSEGYTLYTARDGVSAVGLYADHRPDLIVLDWMLPGMSGLDACRRFREMGEPGIIMVT
AKTEETDKIVGLEMGADDYLTKPFSLRELAARIRSLLRRMNRLPGSAPQSVLRRGELAIHEESLQVFKGGQELRLTPTEF
RLLHLLASKPGRVYSRMQLLQHAFEEEFVIDERTVDSHISKLRKKIGDDPEQPVYIRTVYGFGYRFGAGHEN
>lcl|NZ_CP026520.1_prot_WP_084706692.1_5282 [locus_tag=PC41400_RS30470] [protein=hypothetical protein] [protein_id=WP_084706692.1] [location=5853707..5854225] [gbkey=CDS]
MSYRLSMKQKNWVVVLHVLSIVGWLGGAVCMLLLSRHMLYADQGEQLYYTLENMHLIDSTLIRYPALIALVTGIVLSLWT
QWGIFKYYWILIKLVLTVSLIVFGITCMGQWLSELFRTADEMRLAALGDAKFLDTAYRLLGGAVANIGALVFMTAISYFK
PLGKIRKTARKS
>lcl|NZ_CP026520.1_prot_WP_042235408.1_5283 [locus_tag=PC41400_RS26670] [protein=YafY family protein] [protein_id=WP_042235408.1] [location=complement(5854420..5855373)] [gbkey=CDS]
MSKSRRQIELMITLNTKRRFTAGELAREFGVSKRTILRDLQELTEAGLPLYSEPGAAGGYQMLKEKTLPPILFSENEAVA
MFFAYQSLRDYADLPFQPESLSALKKFTHYLPEETRQRINGLRTRFTFWTPQRHVEAPHLAGLLEQAVEQRIIRIRYDAN
TDIVTRKIQPIGIYATNGLWYCPAYCFERSAIRVFRVDRITAIEPVEDQSGKRNYDKLTVQEYLSLSDEEGASPLHVLLD
KEGVRRCSTETWMAEGLTVYEDGTGTISRMIRPDFIPWTARFFVSFGKEARVEQPAALVAAIRELIAELSVQYPGPG
>lcl|NZ_CP026520.1_prot_WP_042235406.1_5284 [locus_tag=PC41400_RS26675] [protein=DNA alkylation repair protein] [protein_id=WP_042235406.1] [location=5855514..5856593] [gbkey=CDS]
MELLKDLYSPSFFDRFITLVKEKYPSFDGDRFRGSIYDGNWDTEAFKQRMRHIACSLTETLPSSYPEALEVLKAVAPQCR
GVEYLFFPDFIELNGLGSYELSMEFLQTFTPYSSSEFAVRPFLLREPVLTLEHMKQWARHENEHIRRLASEGCRPRLPWA
SNLPALREDPRPILPILETLKEDPSEYVRKSVANNLNDISKDHPELVLKLAKDWYGKSPATNWIVKHGCRSLLRKGDSRA
LSLFGYEEIRSISVHALEVKTRSIAEGEDLAFTFRLVNESEDIQALRVDYEIGYRKKNGTLAPKRFKLTEKPFGPGAHPL
AKRHSFKLLTTRTYYAGEHEVRILVNGTPMAAVSFEFTG
>lcl|NZ_CP026520.1_prot_WP_042235400.1_5285 [locus_tag=PC41400_RS26680] [protein=response regulator transcription factor] [protein_id=WP_042235400.1] [location=complement(5856872..5857471)] [gbkey=CDS]
MIRIVIAEDQRMLLGALASLLNLEEDMTVVGVAGNGEDALKLVQEHKPDVCIMDIEMPAKSGLDAAEELKGSGCKVIILT
TFARSGYFERAMKAGVNGYLLKDSPSEELADSIRSVMAGRRLYASELVDEAYSGENPLTEREKEVLLLIADGKNTKEIAS
QMYITTGTVRNYISVILDKLDVSNRIEAITRFKEKGWFK
>lcl|NZ_CP026520.1_prot_WP_042235398.1_5286 [locus_tag=PC41400_RS26685] [protein=sensor histidine kinase] [protein_id=WP_042235398.1] [location=complement(5857468..5858604)] [gbkey=CDS]
MQNWIRIFQKNTGLSPYVWVVFYILPFYFIFSSGSTLKNVFGIGMIIVFFCCYVLSVVSKGWLVYFWTSVQIVVSISMTV
LFGYVYFSLFLAFFIGNIHNKIGFFTFYTIHLVSTIATINYGFVTQNPVFITQLPFVLISLIGVILLPVNTYTKNKRDKL
QEQLEDANKRIDELVKLEERQRIARDLHDTLGQKLSLIGLKSDLAGKLIAKNPSQARAEINDVRQTARSALKEVREMVTN
MRSTRVAEEMFRVKQILKAAQISFKLEGDPKQIDASLITENVVSMCLKEAVTNVVKHSGATACTIVIEPSRTDLVVKVKD
NGNGRAEQFDYFRGNGLRGMKERLEFINGTLDIVSDGGTTVVMKIPIVLKQPAKEAGE
>lcl|NZ_CP026520.1_prot_WP_042235395.1_5287 [locus_tag=PC41400_RS26690] [protein=fatty acid desaturase] [protein_id=WP_042235395.1] [location=complement(5858670..5859725)] [gbkey=CDS]
MSQTKVMNLKKDVAPYEKTDTKSSIRQIFNTIVPLLLLWYAAYASLSVSYWITLPLTLITAGFVIRTFIIFHDCCHQSFF
KSRRANDILGTITGVLTLCPYEQWKYSHSVHHATSSNLDKRGTGDMWVLTVEEYAAAPWFRKAAYRIYRNPLVMFGLGPI
AVFLIQYRFNRKKAKRKERINTYVTNILIVLLYALMCWLIGWEAFVLVQAPVFFVSGLLGIWLFYVQHQFEDSYFEKEDE
WSYVKAAVDGSSYYQLPKVLQWITGNIGYHHVHHLSPRVPNYHLEKAHNSTPPLQKATTITLRTSLQSLRFRLWDEKGKT
FVSFKDVKHVLRTPGAASDVLKNRKPGFQGE
>lcl|NZ_CP026520.1_prot_WP_042235392.1_5288 [locus_tag=PC41400_RS26695] [protein=DUF1273 domain-containing protein] [protein_id=WP_042235392.1] [location=complement(5859873..5860430)] [gbkey=CDS]
MKNLLVTGYRAHELNIFNEKHEGIEYIKKSIAGRLVPLIEDGLEWVITPGQYGVDLWACEVVIGLKKEYPQLKLSILTAY
ANPEEKWKEDKQEFYRGILKDVDFYGAVSKQPYAGVWQLQARDDLLLRKTDGLLLFYDEEAGEGSPRFMKERALKKQNED
GYAVMIITPEEIQDIADEENRADFY
>lcl|NZ_CP026520.1_prot_WP_042235390.1_5289 [locus_tag=PC41400_RS26700] [protein=anaerobic ribonucleoside triphosphate reductase] [protein_id=WP_042235390.1] [location=5860682..5862661] [gbkey=CDS]
MSLLDRTGLGNADAMLEEIKQWGDAIMEGSDADLLRENANLNGESFSGKMSKFGSEYARWYARRFTMPAILVQAVDANYL
YVHDLDQYVLGTTNCMFIPFAKLLRRGFDTGSGSVRPPNSISSAMALVAIIFQSQQNAQYGGVSANKLDHDLAPYVAKSF
AKHFRKALRYFGENEPVDDERGISMSRADLRVLCPESYRYALEETEEETMQGAESLIHNLNTMSSRAGGQIPFTSINYGT
CTSPEGQLVIRSILTATMRGLGHGETPIFPIQIFKCKKGVNLESGDPNYTLFLKAAECSSRRLYPNFANLDAPLNLEHYD
PADPDTEFATMGCRTRVVADRFGRNRLSGKGNLSFNTLNLVKPGIEYGIVSGRRSKADEAGFYRKLTRYLEIALDGLLHR
FGIQAAQKTKASDFMMREGVWEGGEQLDPDDTVGDLLKHGSLSIGFIGLAECMKAMYGSHHGESPQVYAKAIELIRYLRH
FCDAKSEELNINLTLFATPAEGLSGKFTKRDREAYGCLPGVTDREYYTNSFHIPVYHPMTSAEKIRLEAPFHDLCNAGAI
TYVELNGNARNNMPAFVKIIKFALSRQVSYFSINHPIDRCSACGYEGIIGAVCPACGSHEKDVHIRRLRRVTGYLTGDYQ
TRFNGAKQAEVRDRVKHSP
>lcl|NZ_CP026520.1_prot_WP_042235388.1_5290 [gene=nrdG] [locus_tag=PC41400_RS26705] [protein=anaerobic ribonucleoside-triphosphate reductase activating protein] [protein_id=WP_042235388.1] [location=5862658..5863191] [gbkey=CDS]
MNICGYAPETIDEGEGLRTALYISGCRHYCRGCFSPRTWDFDYGTPMGTDWEERIIAEIQSNPLLDGISILGGDPFFSAG
EVSSFLDRLRQRAGTVNVWIYSGYTYEELAANPESVEFGLLSRCQVLVDGRYVEELRDPALLYRGSRNQRLIDIGSSLRH
RKAVLWQSAWDREPPTA
>lcl|NZ_CP026520.1_prot_WP_042235386.1_5291 [locus_tag=PC41400_RS26710] [protein=metallophosphoesterase] [protein_id=WP_042235386.1] [location=complement(5863327..5865912)] [gbkey=CDS]
MNIQTRVHTIFMLVGSTECGKTTFAKEVLIPGLQFEASDKAVKANVQYLSSDLIRQEILGFDYDKYDQIMLESSGQAFHL
LFERLKMATSFPVNAEFVVVDTTGLAEDFREKVRAVAKENNYNLEVILFDYRKRHDYYASERSKKLITSHINRLKKDVMG
SLSREGYTKIHKVRAKDFYSPDTGEANGAYTVEILDLDEYLAAVLPQGYDCIIVGDVHECVDDLKGLLLDYGFKLSGNSL
ELPEKLRHTKIVLTGDWIDKGHRTRAVVEFLYENREHFLFVLGNHENFVFKYLKGEIGGVEEELFHSYFTSVETLRKDAE
LLRKFNELVEISKPFYRKVGTRGPSFYVTHSPCRNKYIGKLDANSARHQRNFRLDREESFEKQLAFVKEEAVRNHPYHVF
GHVAAKNAFRIGNKIHVDTGCVNGNKLTAVSLSFKPFFKSRSAEASVLPEDLPVLFREERKVTLQDLDAEELRRLHYCSR
NRVNTISGTMSPADKDETAGELESLRQGLRYFADHGVHQVVLQPKYMGSRCQIYLHRDPDHCYGVSRNGYKIKGADLSDI
FRSLLDKFDSYMEENKIAVLLLDGELLPWHALGEGLIERQFRPIEKALEAELDFLKANGFEEAFGKLAEAYTASGFEKDQ
FTMPKARLSEKYGASVYQNYKHMRGLLEKRVPLEEHEAAYRTYKRQLELYGEAGEPEYKPFAILKLIYENGEEQLPQMTT
SEAYGFLSGDEFLLLNLGDPDSYEKAEAYFKSLTTDFHMEGVVIKPELPGSETVPYMKVRNGDYLTLVYGYDYRFPHKYG
KLMKQKNIKLKLRTSLNEYRLSKEMLAVKYSDISPDNEAYKKAAANVLFEVAKEKEIDPRL
>lcl|NZ_CP026520.1_prot_WP_042235384.1_5292 [locus_tag=PC41400_RS26715] [protein=class I SAM-dependent methyltransferase] [protein_id=WP_042235384.1] [location=complement(5865916..5867181)] [gbkey=CDS]
MAIVQLSSTNPQFSYLIRKNPESGMLLRPVRKGIAYGWYTGENTFNVYFKDADNEISYKQHDKEEFEYLNVSRYNTPLFP
LNAWNEFFTAPLKIRDERDSEGFTHTVRIPMIHIEMIRYIHFFEKHLPDFSFEIAHQAHKTYSLTVTTGLSLYHLLHAVS
VLCLFLSTFGNEYIDISELLLEKYIRSVQVIDAPFYIRSLFVRNFLNSRERFKKYKGELEKTDRYEIDFDFGGTGHQRRT
FIRGQLPFDKAVLDIGCGEGFYAIPFAAKLEDTYYAVDIDEAALETVTRKARAKELDNLVPLPSIRHFLDQYNGEKVDVI
LTEVIEHMSPEEAQALILQVCAEVDFDRFIITTPNADFNPFYELDGFRHPDHKWEMGEKEFGQWIDGFIGAAGCRAERAA
IGDGVNSIRTTQGVILTKKEA
>lcl|NZ_CP026520.1_prot_WP_042235382.1_5293 [locus_tag=PC41400_RS26720] [protein=WYL domain-containing protein] [protein_id=WP_042235382.1] [location=5867355..5868686] [gbkey=CDS]
MARESFDKEIQFLRLLVLTSGAYSRQQFAERLGISVHTFDKTLRRLKEIASTVSYPLAEEQGREFAEMLRFNYYESSDPL
LLFLYRAKSMKESESRRLPFLLAVMQEQDQAMSAGELIDACCGGMPDDVPLPDEKTIRSDLKYLEEVGVIRKEGSGRPYR
YRLHNDLIRQLSREELLDLYDFADVMANTRLPSVQGYLLRDSLKKAIRQQGLETAIAEPFLYKYHYYSRILDEAHLFPLF
QAIRQRRKLRFLYFSPKTHKSYASRNTNPLFERETNGKQAHVLPLKVVYDHQYGRWYLLASDARQGIMKFRIEGLTQIEE
QNPVPEEEFRRKLAELEEKTRYSWLIDTSGPVKVRARFYRPLDAPADFIKERVLLQGQWGTITEENEESFVYEITVNGLT
EIKPWLRSFGSSCEVLAPRQLRKEFIAEWKEIHAYYEPVRENL
>lcl|NZ_CP026520.1_prot_WP_042235380.1_5294 [locus_tag=PC41400_RS26725] [protein=WYL domain-containing protein] [protein_id=WP_042235380.1] [location=5868661..5869542] [gbkey=CDS]
MNLFEKIFNYQMISRLEESGVVALTSHERTWLKSVLKHPAAEEAFAAETLAKLRGLLEEEADLETDSLVTEKAKSPDKQV
YHPLLRPLRRFMRERQGIRLSYRIRNGRTMESQSGIPYKLEYSMVKKEWYLLWYHLRNRMYMATKLGSILELEAEAPADG
VYEEITAKLDQSVGLRKESSVIEVVRTYNRELSRILYAFSCFEKEVSYDDENDLYRIRLTYMREDAEYILSKIRFLGMRV
KVVEGRHLRWRMADSSARALARYGDEPEENGGTSPDAEMEKHIETPKASAPGS
>lcl|NZ_CP026520.1_prot_WP_241688742.1_5295 [locus_tag=PC41400_RS26730] [protein=class I SAM-dependent methyltransferase] [protein_id=WP_241688742.1] [location=5869668..5870441] [gbkey=CDS]
MADMIPSGKGVVRKMNETVYDPDFFKSQQEGSRKSACEIVPILLDLINPESVIDIGCGTGAWLSVFKDLGVREILGVDGD
YVNEDMLQIPKENFRKHDLAAFFSSGKQYDLVVSLEVGEHLPHASAETYVNTLVSHGDAVLFSAAIPYQGGKNHINEQWP
GYWAELFNKKGFVAVDCIRKQVWENAHVQWWYAQNTLLYVREEALSRYASLKDYQPSESDSLSLVHPRLYLHLAKDFFGD
DGYKDGYIIGKKFTPRI
>lcl|NZ_CP026520.1_prot_WP_042235378.1_5296 [locus_tag=PC41400_RS26735] [protein=glycosyl hydrolase family 18 protein] [protein_id=WP_042235378.1] [location=complement(5870547..5871686)] [gbkey=CDS]
MKIHLYTKLFVALLISGMGFYFYTTSQAKDHKINSVYVIPTGDKLDEINFEEKKIDIAFLAFAQIGDKYKVSFHQDEAAD
AEIKENIKKLKKNNPKTSIVLAVGGYGADGFSDASLQENRRTFTDNIIALVKELGLDGVDIDWEYPASDAWGTQKSRSED
TQNFTFLMKELREKLNQLPHKNKKYILTFASGTQDWYFKNVEVKKVEKYVDYINVMSYDLTGKWSDTTGLNSNLYKDAEN
KSIDSVDSIVKLYLNHHIDPAKLLLGVPAYSYGWEDVKSDGDGAFSPGKPIDIDKTDLSYKTIEKKYMDKNGFKRYFDEK
AKAAYLYDGETFISYEDKEALREKIAYIKANNLGGAMVWEYSQDADDGLVKYLGDHLNE
>lcl|NZ_CP026520.1_prot_WP_042235376.1_5297 [locus_tag=PC41400_RS26740] [protein=acyltransferase family protein] [protein_id=WP_042235376.1] [location=complement(5871689..5872792)] [gbkey=CDS]
MNNAVLKTNRIIYIDLLRILSILAVIILHITATLLTSTNDFNTASWWVSNLFNSAARFAVPVFFMISGAMILRSEVKSYK
TFFLKRIMPLFVSLVSWSLIYSIFTQYYLQKSTMGAGEFLLSFAYKFILDRNYIHLWFLYAIIAIYLTVPLISRMVKACT
EKELRYYLLLWFTVSIAYRFITDCVLRITSESLYVPIMNIPLFMGYLGYFILGYYLFHYEISPKLKNVLFNLGIVSFFLT
PVATYFASMYSGVLDEMFYGNYSVTTFFMAAGLFILFKEKDAAISAKVNDKVKRLVGSISKASFSIYLIHLLIEIMVSRR
TEVEGSLVETTITLLFNLGAVFVVSYILVKILNLNKIVTNVLFGGKG
>lcl|NZ_CP026520.1_prot_WP_042235374.1_5298 [locus_tag=PC41400_RS26745] [protein=glycosyltransferase family 2 protein] [protein_id=WP_042235374.1] [location=complement(5872808..5874184)] [gbkey=CDS]
MQILLYVTTIVFLAFQLLYTFIPLLFSKVKKLEEDLEEKSISVLVPAYNEELTIRNCIEAMEGLKYRNHEIIIINDGSKD
ATLRRLQELLDLQEDDRQVMKELTYQPVRAFYRSRQYPHIYVIDKKNGGKADSLNAGIDYALSDIVITLDADSMLEAHSL
QYVNQYFHDPDVMALGGTVKIVQGAERKDGQIQERFTGKGLIKSQIISYIHGFYVRKLTQSALNSIVVISGAFGAFYKEV
LIQVDGFRSTVGEDIDITLKIHEYIKANNLSKKLVYAPEAVCYTECPENMPNFYKQRIRWQKAFVDCILIYWSDLPRKFN
LGVRLFFAIDGFVLGTLTAFTTFVYLFQLLLAGDGIWMALLILSATLLINAIQIVIALALCKKYGSTYTLKDYIRMFIFS
QLELLTYRNVLLYINIVGTFKYFDNDEGWGYVERKGIASISRSVTAPPSAPASPLSQS
>lcl|NZ_CP026520.1_prot_WP_042235372.1_5299 [locus_tag=PC41400_RS26750] [protein=phosphodiester glycosidase family protein] [protein_id=WP_042235372.1] [location=complement(5874572..5875234)] [gbkey=CDS]
MAYTFSTVTASNGVKLYSIKTSPKNITIKSIGASSVVGQTAYGVNGGYFYDGTLLQIAVNNNVAVAPNGSENAKYARGTL
VWDDQSGGFSVQVVQKMSEIQVANKSNYWAQGGISMMLKSSDSAWKTQAAAENIQNPDGAANRTALVYNNANNIWLIVTD
SKPTAAAFRTAIKEKIGSGTLLDGIFLDGSGSSQYKVAEGSYGGSDAKPRLVYQMVALKS
>lcl|NZ_CP026520.1_prot_WP_042235369.1_5300 [locus_tag=PC41400_RS26755] [protein=urea carboxylase-associated family protein] [protein_id=WP_042235369.1] [location=5875430..5876020] [gbkey=CDS]
MSTQKWRIPATEGLGFRLGKGQTVRITDVEGEQVADIVAYRADNPTEKFDPTVTMDALRKMKVTPGDILYSTRYTPLLTI
VADTVGQHDFINSACRPEMYEFTYRKKDHACCYHNLNKAIAQFGIEPPEQHYPFNLFMHTVIQPDGSIRVKRPLSKAGDY
VDLRAETDLLVAMSACPASESICNGYVCTPIDIQIL
>lcl|NZ_CP026520.1_prot_WP_042235367.1_5301 [locus_tag=PC41400_RS26760] [protein=LysE family transporter] [protein_id=WP_042235367.1] [location=5876103..5876729] [gbkey=CDS]
MNAFLTYVVLGISLSAPVGPVNAAQLEKGVRFGFLHAWLVGVGAMIADLFYMLLIYFGVAHFLDTPFMKTFLWLFGCFVL
LYTGIVTLKSLKLNESGEMRESPSEISSFRFGFFMALTSPLTILFWLGIYGSILAQSPQSHDPGQALWHSLGIFVGLMIW
DIAMALFASSFHRFGNPLILQITSLLAGISLILFGLYFGYEALLMLAG
>lcl|NZ_CP026520.1_prot_WP_042235365.1_5302 [locus_tag=PC41400_RS26765] [protein=YqcI/YcgG family protein] [protein_id=WP_042235365.1] [location=5876825..5877589] [gbkey=CDS]
MAELRSKIWIEEHGMELPEWQRTAFGEFAAMIAGPENKYPCVPGHQGYVTNNLRYGFAPDPRTAQAVSDTAELLNAYGSC
SRETGKYASLVIFFETPEDLKKHADVLEYETLFWSLLKRLHDKDTAPWPENIPQDPAKPEWEFCFAGQAYFAFCATPAHG
QRKSRHFSSFLVAFQPRWVFNEINDTTAFGRNISKIIRDKLADYDDVPVHPSLKWYGQPDNQEWKQYFLRDDETAPSRCP
FMAMKNQLQKNAPF
>lcl|NZ_CP026520.1_prot_WP_042235363.1_5303 [locus_tag=PC41400_RS26770] [protein=spore coat protein] [protein_id=WP_042235363.1] [location=complement(5877646..5878302)] [gbkey=CDS]
MQQQNYFNNQNQQGLPNFNHGGHEFFDMHEVLSGTINVLDQFMLFRTFVKDPELLDIIDRQYSFILSQYNLTAECFATGR
KPSHETSTYMMQQPTNIQYGIKPSQPKKPNQSLADVKDAGISGHMLGLIKTHASLLTMSAAEVTNPVVRRVLASQVQNFI
ELAYEIFLYQNKHAYYQVPQLQQNDMQSMLNTFVPTPGYPQSPIGNQGIPANFAPRYQ
>lcl|NZ_CP026520.1_prot_WP_042235360.1_5304 [locus_tag=PC41400_RS26775] [protein=glucoamylase family protein] [protein_id=WP_042235360.1] [location=complement(5878399..5886975)] [gbkey=CDS]
MTYTNEQLSQKARELALQHVTTLPRTRPAYIEQHFKKDIESLRDFVLSLQTVHAGCTQPAEQWLLDNSEFVEEQAQEVRA
LLLQDALDRMPRLRGSGLLRIQSMMEDYLTHVEGILSEETILAYIQAYQEVSALTLTEARSVPLILRLVLVKKLAETMAF
VRERREVCIDVGRILSLIEPAALSPETVHLQVEKAGLTLPLPGPWIVHLISHLREWAGDAASVREWLICSYENGAEDLDR
IVSYEHRLQTAYQARAGSVITSLRKNERWDWNDLFEQISLPDRTLRQEYTGVYPLLDAYSRNQILAEVERLARRLRVPEN
LVAGQAVALAREAAERADAARAGSGSPGAASGHADASARSEVLILHENGEVAAPLAAGDRTRQQFPSYYLLESEGVQKLI
ASLRKCSSPRTMPENAISRRRAGTYFASMLGLFAVLWLIAAAWIGIGHGLSALAWAVVLAALLLPVSEWGMNWLHFGIER
ICRPRPLLRYDFSAGVPAEAGTMVVIPAIWSSPDEVAELADRLEVHYLANRDSNIHYALLGDFTDAPDEKLPEDDRLVRL
AEDKIRALNEKYSAAGGTTFHFYQRRRLWNPHEGVYMGWERKRGKLVEFVELLRGSGDTSYAVKFGDASVLPRIRYLITL
DADTQLPMGSAQRMIGTLHLPYNRPRLNQARTRVVEGYGVLQPRIGISHDSAMRSRLAYFWSEPGMDPYAFAASDPYQDA
LGQGIFTGKGIFDVDVFAELLCERIPDNTVLSHDLLEGGFMRAALLSDIELIDDHPAKFVSYQKRLHRWVRGDWQLVCWL
FSRICDRRGVLRPIDLSFLTRWQMLDNLRRSLLPPAVFVLLVLAWTVLPGSPWRWAAVLLATMFLPLLRQLFILPRIYKQ
PRRLLATALHVMIQIWTLPFQTAVLLDAIGRTLYRVFVSKRRLLEWTSSSHIERSTRGQQRTPLLQAGWGYVLIIVFALA
SAVQASASLLWAGLALSLFWALAPLAVSWLDRPVEREEAPLGAEEQEKLRVLARQIWSFYEDYAGPKDHYLPPDNVQVDP
PNGVAHRTSPTNIGFLLTSALAAREFGFIDTPSLVEKLENTIGTVENLEKWHGHLYNWYDTLTLGILPPAYVSTVDSGNF
VASLMTVKQGLASWLGQEAGTEGRYMEINRAPAGAYGVEFAVDLDQTPRGESALNGIDGNTEGKGEDPDNSRNIGPKSGS
KSGSGRSAVLTAEGSGPEHAGHEAPSNLPDWLTRGQQLVQRLESLIAATDFTQLYDHKAKLFVLGYHGATGERDRILYDL
LASEARQTSFVAIALGQVSVSHWMALGRTVKLQGNKTTLISWSGTMFEYMMPWLIMRTYRNTLWDSTYRGVVKRQMEYAH
ERGVPFGISESGYYAFDYQMNYQYRAFGVPGLGFKRGLEEDLVIAPYAAVMALPFALREGLEDLNRMEELGARGKYGFYE
AIDFTADRMPQGEDCKIIRSFMAHHQGMSLLTIANLLLPVKMYDYFHSDKRVQAAELLLIERIPPRESMLTRELPGKSRI
QKPEAERTYSAPESIAADTPAPEVGVHSNGTFTTAVTNSGSGFIRWNGLAVSRWREDPVGDPWGVYLYIRDVTREKVWSP
AFQPCRVPSDSQSVLFSQERTTFRREDDGIQTVLEITVSPEMNAEVRRLTLENKSSEARIVEVTTFLEIALALPDADKAH
PAFTKLFVETQFEEETGCLLARKRPRNAGEKSIWAFHNLSAFGAVLGPAEFETDRAAFIGRGHSLARPKGLASRLEGTVG
SVADPAFVMRRRISLKPGESIRLCAVTGVADSKEQSLNMVRQLFQAQQVDRTFQLAWTRSQIDLQHLHITPAEAAVYRAL
AGRVLYRGPFKPEQAESIAANSKGQQGLWAYGVSGDRPIVLARVADSANLPFVQKLLGGFEYLRRHGLFFDLVLFNESPG
GYQQELRDALVRLSEQAAGNAVAAGQGGIHVVPASAMPQEDHALLLAAARVVLRADGPSLRAQLKIWPRKDARDVPLPVT
AGEAPSPLRPDASAVSGTAGAAAEASGAAGTTGVSGINGVEAAGAVPAADTLFFNGWGGFSPDGREYRITLKNGEHLPAP
WINVMANPRFGCLASELYTGYTWWRNSRECKLTPWSNDPALDPPGEVFYIRDEENGDFWQMAPSRRQDDPSVYRVAHGRG
YSRYEHESRGIRQEMTVFVPLEDPVKIVEVKLRNTTSHKKSLSATYYAEWVLGVQREGNASFIMTDWDDEADSLIARNAY
QENFRDATAFLTVHADESSLSWTGDRLEFIGRNRTFEDPAAMSRPDLSGVTGPVYDGCGAVRTKFELEPGAARTVYILLG
CGDSREEAVRIARKYRESDACMAAYEAVRTFWEDLLGTVTVSTPVPEMDMMLNHWLLYQTLACRMWARTAFYQAGGAYGF
RDQLQDSLALLNSRPDLTRAQILLHASHQYEEGDVQHWWHEETHRGIRTRFSDDLLWLPYAVSRYIDHTEDRSILDETAP
FLTSKPLEADEHERYEETVLSGNSDTIFDHCVRAVDRALKFGEHGLPLIGIGDWNDGMSMIGPEGRGESVWLGWFLGDVL
QGMAELCEIRGKQDHAVRFRSQREKLAAALNESAWDGQWYRRAFTDAGQWLGSIHNAECRIDSIAQSWSVISGMAPDDRA
RQAMKSFDRELVDRSLSVAHILTPPFDKTDPSPGYIQGYPPGIRENGGQYTHGVIWSIVAWCGLGNGEKALELFQMFNPI
MHTKTAAEVRKYVGEPYAMAADVYTEPPHRGHAGWTWYTGASGWMYQAGLEWILGIRRRGAKLSIRPCIPPEWPEFSAVY
RFGKSEYRITVLNPSRKSSGCMRLTIDGQEADGDFSSGEGPFVPLTDDNRVHEIVVTL
>lcl|NZ_CP026520.1_prot_WP_042235357.1_5305 [gene=htpG] [locus_tag=PC41400_RS26780] [protein=molecular chaperone HtpG] [protein_id=WP_042235357.1] [location=complement(5887254..5889134)] [gbkey=CDS]
MAKKQFKAESKRLLEMMINSIYTQREIFLRELISNASDAIDKIYYKALTDDNLAFNADNYFIKLKADRAGRTLTLSDTGI
GMTKEDLENNLGVIAKSGSLAFKKENEAKEGHNIIGQFGVGFYSAFMVADVVTVISRALGSEEAFKWESKGADGYTIVPA
DKETVGTEIILKIKENTEDDNYDEYLEEYRLKAIIKKYSDFIRYPIKMDVSGRRPKEGSDDEFEEYQEEQTINSMVPIWR
KNKNELTTEDYENFYAEKRYGFDKPLKHIHISADGAVVYKAILFIPESTPFDYYTKEYEKGLELYSNGVLIMNKCGDLLP
DYFGFVKGMVDSEDLSLNISREMLQHDRQLTLIAKNIKNKIKSQLQGMLKDERDKYEQFYKSFGRQLKFGVYNDYGTHKD
DLQDLLMFTSSKEKKLVTLDEYVSAMPEDQKYIYYASGESIDRIEKLPQTEMVLDKGYEILYFTDDVDEFAIKMLMAYKE
KEFKSVSSGDLGIEPDAADKQTEEEENDSKDLFEAMKAILSDKVKNVKASKRLKSHPVCLSTEGELTIEMEKILKAMPNG
QEVQADKVLEINVNHDVFCSLKEAYASDKDKLNLYTNLLYNQALLIEGLQVSDPVEFTNNICKIMV
>lcl|NZ_CP026520.1_prot_WP_241688743.1_5306 [locus_tag=PC41400_RS30475] [protein=S-layer homology domain-containing protein] [protein_id=WP_241688743.1] [location=complement(5889334..5893809)] [gbkey=CDS]
MHQRKSSFRWISVMLMSVIVLNAVLFGAAGPVKAAPPGGLSAYNAAVTAAEMYQAILDYPNDIFETVQQKDSFYWNRKAD
NDAIVDDLIGKRPFATVTDLRDAYGSALDNRNFFLFLHEATDNSGVSAALSWVNDGTFLNPGLDLSKYFALDYTEQAVVD
QAILDGIPVDGYAAASEIQTIIDTTIGQIVSGALQDDKTALEIGYAVGDSDAHVTQNLYLPDAGTHFTQITWTGGNGVID
PATGIVTPPAGGDVTAALQAKISRNNESVTKDFTLTVKSSAPPATPPAITAHPQSVPPILPGMPASFSVTATGTGLTYQW
QVNTGSGFVNLPGDTNATLSIPAVSPSMNGYMYRVIVSGTGGQPVTSNGAVLNVMPPIPTFPPAITAQPQHVLPPVLEGN
PASFSVTATGTGLTYQWQVNTGSGFLNLQGETNATLSIPAVSMGMNGYMYRVVVSGTGGQPVTSNGAVLNVMPPIPTFPP
AITAQPQNTPPLLQGMPASFSVTATGTSLTYQWQVNTGSGFLNLPGETNATLSILAVSPSMNGYMYRVIVSGTGGQPVTS
NGAALNVMPPFPTFPPVITLHPQSPPPVLEGNPASFSVMATGMGLTYQWQEDSGIGFLNLQGETNATLSIPAVSLGMNNY
KYRVVVSGTGGQPVTSNSAVLTVLPMIPTLPPAITAQPQNTPPLLQGMPASFSVTATGTSLTYQWQVNTGSGFQNLPGET
NATLSIPAVSPSMNGYMYRVIVSGTGGQPVTSNGAVLNVMSPIPTFPPVITLHPQSPLPVVEGNPASFSVMAAGMGLTYQ
WQEDSGSGFLNLLGETNATLTIPAVSMGMNGYTYRVVVSGMGGQQVTSNGAVLNVLPPIPTFPPAITAHPQNVPPVLEGA
PASFTVIATGTGLTYQWQLDTGSGFQDLQGETNATLNIPVVSQGMNGYKYRVVVSGTVGQPVTSTEAVLTVYTVPSEPTN
VTATAGDGQAVVSFTAPVSDGGTAITGYEVTASPGDKVVSGTASPITVTGLANGTAYTFTVKALNAAGGSAASVPSNSVT
PYAPSSGGGGGGGSSEPSTPTPTTPPVTEVPANEGADILVNGKTERAGTVKSTEVNGQSVLTVTVDPKKLAEKLASEGQH
AVVTIPLAKDKKADVLVGELTGQMVKSMEQKQAVVKISFGDVTYTIPAEQINVSALSDQLGKPSALEDIKVRIEVAAPSK
ETLKVVENSAAAGNFTLVAPALNFTVKGTYGGATVEVSRFNAYVERTIAIPDGVDPGKITTGVVVEADGTVRHVPTRIVN
VNGHYFAKINSLTNSTYSVIWHPIEFRDVAGHWAQNAVNDMGSRMVVSGVENGTFNPDTAITRAEFAAIMVRALGLKQEK
ASAPFSDVNASDWYSAAVQTAYAYKLIGGFEDGTFRPSDKITREQAMVIMAKAMKTTGLRDKLPSKEAAELLAPYADAAA
TAEWAKSGVADCLQGAIFAGRTATELAPKASITRAEVAAIVQRLLQKSELI
>lcl|NZ_CP026520.1_prot_WP_042235842.1_5307 [locus_tag=PC41400_RS26790] [protein=response regulator] [protein_id=WP_042235842.1] [location=complement(5894105..5895238)] [gbkey=CDS]
MKAILIDDEKPALIQMERLLRTDGRMEVTGMYTQARHGLEHMTAEKADIVFLDIEMPEMNGLEAAENLYRLNPEAQIVYV
TAYNDYAIEAFELGAIDYLLKPVDPERFAKTVSRLESRLSQLQAVRREAAPSGEPSVLCFKQLSLTGGPGTGGSLRWRTQ
KAQELFAFLVHHRSKRVAKDVILETLWPELSLDKAVTQLHTSVYQVRKMLKAWGIGAGIDYSHDSYRFTGDGFRTDVDLF
LTAASSGEITSREQWRRLDQALQLYRGDYLEEHDYFWAKSYRAQLLGQYIQASLLASAYEVSAGQTHSALRRLELLQKKE
PFSEEICRAGMEVHAGLGDKLSLKKHYEAFASALREELGSEPGRETTNLYRHLMQSI
>lcl|NZ_CP026520.1_prot_WP_042235845.1_5308 [locus_tag=PC41400_RS26795] [protein=ATP-binding protein] [protein_id=WP_042235845.1] [location=complement(5895219..5898356)] [gbkey=CDS]
MKKQALFFILGAALILLLPVYLVLQHVVSIERNPDAVDGTLDLRAWNFADKGTVRLSGEWEFYRGKLLVPGDFHPSAETG
KTPPMPTGMAQIPGKWNAYMGEEGDPAAEGYATFRLNVRLNETDGAIYGIRTGNIRMANRVFVNGLEVGASGTPGRTAAE
ENSDNVPYTGYTYLKGSEAEIVVQVSNHSYSSGGIFTPILFGDEKSVMKSREAALFSEWMTMAGFLIPGLYFLVLYRMRR
NDPASLYLGLFCLVGLLFVLTHGEKMLAAAWPGFSYTWLLKVQLMVSSFVYFFLLRYVAVAVPGGVHPLALRVSSFVTAA
GLLAGLLLPPVVFSRLEVLILAISFLSMCYMAFVLLRDVRKRSEDTLLMIAGVMSILMVIAAYVLNLVGLYEGLLLVPAE
MLMFVLTQAILLAQRFNRTFVEVDQLSQRLLSLDGLKDEFMANTSHELRTPLHGMVNMAQSMLEGASGALNVKQRNDLAM
IVSTGRRLTLLINDILDFSKLKNGEIVLKRQPVDLKAATNAVLEVMTHVARHKDIRFEQEWPDRLPWLLTDEDRLQQILY
NLIGNAVKFTPAGTVTIGAVTKNGHVLVFVKDTGIGIAENRMEEIFHAYDQGVDPGDRDYEGTGLGLSITKKLVELGEGK
IEAESRLGEGSVFRFTVPSTTAAPEAARTTGGGFRASVARDETAAAVLPEGLEDRRAGKIDATVLLVDDDPVNLQVLYNL
LSVEGYRLIAVDGGAKAMAHIGSGLSIDLVIADWMMPGMSGLELCRAIRDRISLAELPVLLLTARSLPEDVRTGFQAGAN
DFLRKPVDADELRARVRTLLEMRKSVRNAVSAEMAFLQAQIKPHFLYNALNTIIALLPSDPDKTTKLLLELSRYLRGSFD
FQNRDQLVPLQKEMALVQSYLVLEKARFEERLQVIYDIRAEPNRLIPPLSIQPIVENAVRHGIMRRAEGGTIGISVEEKE
DVLLVEITDNGVGMEPGRAASLLSAEAGGGVGLINIHTRLLTMFGEGLDIRTEAGQGTTVQFRVPNNGGTRGKREEVKSS
EGHSD
>lcl|NZ_CP026520.1_prot_WP_042235849.1_5309 [locus_tag=PC41400_RS26800] [protein=hypothetical protein] [protein_id=WP_042235849.1] [location=complement(5898827..5899261)] [gbkey=CDS]
MKKNLILLVTVLLLMSCSAARIPYDDKITDRLMASVQKIETGQTVEPDNFEAYDWVYIIPPYSGGEALDNLEVDEQSKKY
IQQQASMNENYILVTSKEGKAVSYATLDKNFSTEGGEILKYKGSDKLLVRKGETGGRPYLFLPK
>lcl|NZ_CP026520.1_prot_WP_042235853.1_5310 [locus_tag=PC41400_RS26805] [protein=acetyl-CoA C-acetyltransferase] [protein_id=WP_042235853.1] [location=complement(5899426..5900610)] [gbkey=CDS]
MTREAVIVSAVRTAIGSFQGSLAGVPATELGSIVIREALERARLEQENVHEVIMGNVLQAGLGQNPARQAWLKSGFSDGV
PALTVNKVCGSGLKAVMLAAQSVMLGDSDIVVAGGMENMSAAPYLLEGARSGLRMGDGKLVDSMIRDGLWCAMCDIHMGE
TAENVADRYGLSREEQDEFAAYSQQKAERAAADGRFRSEIVPVPVPRKKQDPLPFDRDEFPRAGTTAETLSRLRPAFRTG
GTVTAGNASGINDGAAAVVVMSAEEAGRRGLKPLARIKGYASAALDPAVMGLGPVSAVRRLLNRTGVPIDGIDLFEFNEA
FAAQSLAVGRELGVPPEKLNVNGGAIALGHPIGASGARVLVTLLHELERRGGKRGLASLCIGGGQGVALLVERD
>lcl|NZ_CP026520.1_prot_WP_042235856.1_5311 [locus_tag=PC41400_RS26810] [protein=3-oxoacid CoA-transferase subunit B] [protein_id=WP_042235856.1] [location=complement(5900617..5901291)] [gbkey=CDS]
MSHPRQRIVERAVQEIRDGMYVNLGIGMPTLVADEIPAGITVMLQSENGLLGIGPYPKRGEEDPDLINAGKETVTAVPGA
SYFDSAESFAMIRGGHIDLAILGGMEVSGGGDLANWMIPGKMVKGMGGAMDLVSGAKRIVVIMEHVNKYGESKVKKACTL
PLTGRGVVNRLITDLAVFDFTPEGMVLAELLGEAELDEVRAKTEAAFTLGHLLKERLEACGQEG
>lcl|NZ_CP026520.1_prot_WP_042235859.1_5312 [locus_tag=PC41400_RS26815] [protein=CoA transferase subunit A] [protein_id=WP_042235859.1] [location=complement(5901288..5901998)] [gbkey=CDS]
MSKVLKSAEEAVHVIGEGSTLIVGGFGLCGIPEHLIAALERQGTGGLTVVSNNCGVDDWGLGLLLANKQIKKMVSSYVGE
NKTFEKQYLSGELEVELVPQGTLAERIRAGGAGIPAFYTATGVGTLVAEGKEEKHFGGRPYILEKAITGDFALVKAWKAD
TAGNLVFRKTSRNFNPLAAAAGKITLVEAEEIVEAGELDPDDIHTPGIFVQHVVQGGSYQKRIERLTVREAKEETA
>lcl|NZ_CP026520.1_prot_WP_042235861.1_5313 [locus_tag=PC41400_RS26820] [protein=3-hydroxybutyrate dehydrogenase] [protein_id=WP_042235861.1] [location=complement(5902036..5902821)] [gbkey=CDS]
MTPFLAQKTALITGAASGIGLEIARTFAREGAKVVISDIQEEKAKTAARRLADEGYEATGLACDVTDEGQFAGAIRTTHE
TYGRFDILVNNAGLQHVAPIEEFPVEKFEFMIKVMLTGAFIGIKHALPIMKEQQYGRIINMASINGVIGFAGKAAYNSAK
HGLIGLTKVAALEGAAHGITVNALCPGYVDTPLVRGQLEDLARTRGVPLESVLEEVIYPLVPQKRLLSVDEIAEFAMLIA
SDKLKGVTGTSMLIDGGYTAQ
>lcl|NZ_CP026520.1_prot_WP_042235891.1_5314 [locus_tag=PC41400_RS26825] [protein=GntP family permease] [protein_id=WP_042235891.1] [location=complement(5902962..5904341)] [gbkey=CDS]
MFVQVVAIVLALGLLMFFAYRGYPVIVFAPIFTLLAVVISGLALMPSYTETYMANAANYIKTYFPIFLLGAIFGKVMELN
GAASSIARSIVKVLGSKRAMLSVVLACAVLTYGGVSLFVVAFAVYPFAVAVFREADIPKRLIPGTIALGAFTFTMDALPG
TPQVQNIIPTTYFGTDAYAAPVVGIVGGVMVLAAGMFWLEKRRKQAAAAGEGYGTGHKNEPETAEGQRYPNLWVSLVPLV
LVLVFNYLLSRGFLTVNTWYSPELLKSFNIANVKTVASSWALIIALFIGIAAALCINVRHVQSKLAAGLTAAAMGSLLAI
FNTASEVGFGNVVKTLPGFKAIQNFILNISSQPLINEAVSVNILSGVTGSASGGLSIALEVMGKQYLAAGQAAGISPEWL
HRIASMASGGMDTLPHNGAVITLLAITGLTHRQSYKDIFAITVVKTLTVLILAAVLSFM
>lcl|NZ_CP026520.1_prot_WP_042235864.1_5315 [locus_tag=PC41400_RS26830] [protein=LysR family transcriptional regulator] [protein_id=WP_042235864.1] [location=5904552..5905445] [gbkey=CDS]
MDVRNLASVLEVAKQQNFTKAAEILHMTQPTLSKLVKSMEDELGVTLFDRSGKTVKLTDSGAAAIQRMQVILQSVNDLYT
TLDDVSQLKTGTIRIGLPPVISTVFFPRVIAPFQRKYPAIQFVMIEEGAKTVEQRVLDGSIDMGVVISPDEDSSLGRLPF
IRQHVSLVLHRTHPLAERETVSLADLSREPFLLFAKGFAVRHHVLDACHSAGFHPAIAHESSQWDLLVEMVAGGLGVSFI
PEPITSKIANPDVRVIPKTSPGIPWNLYVLWNKERYASYAMRELLEFIGEKAEGAEV
>lcl|NZ_CP026520.1_prot_WP_042235894.1_5316 [locus_tag=PC41400_RS26835] [protein=response regulator] [protein_id=WP_042235894.1] [location=complement(5905605..5905955)] [gbkey=CDS]
MRTLVRMFLEQMGHTVIGEAQNGKEAVDLYGRLKPDLVTMDITMPEMDGLEAVKHIKSANPDAKIIMCSAMGQQQVVMEA
IRSGASDFVVKPIQKERLQEAVEKLFAGQGMKIGVM
>lcl|NZ_CP026520.1_prot_WP_042235866.1_5317 [locus_tag=PC41400_RS26840] [protein=GNAT family N-acetyltransferase] [protein_id=WP_042235866.1] [location=complement(5906106..5906927)] [gbkey=CDS]
MNTGQGQGQTARLIRDNFIAKVSHLASRAEGCTLKQTPEYIRADSGLPSDTFNVAVLLQPRLAAENEAILEETVHYFTDK
KYPMALWSWESKDSVTFETLRQAGLKEAETNIAMYADLDALHPGTECPADFQMKEIASPAEAEQFGDVLVSLFGESEEGA
AVRAFYRKISRLPLWGSSLMKLYIGSYRGRPVSTGTLMFTDDSVGIYDIATRSDFRGQGFGSAMFRFLVGEAKKGKPRLC
VLQASPDGIGIYKRAGFEPVCEVTVYENRHLVE
>lcl|NZ_CP026520.1_prot_WP_042235872.1_5318 [locus_tag=PC41400_RS26845] [protein=hypothetical protein] [protein_id=WP_042235872.1] [location=complement(5907497..5908759)] [gbkey=CDS]
MKTTKLLLVFLVTLALFSNSLPVSATNSIELEASTVYNLDSKGAKLSQLDANSKAVFKNVEITTSKLLLSGNLVNGKLFD
LQGTIKKSENSNADLLVGDLKDSLNNFEVIYFGIDRASNSSLTLNKNKFKGGEAVLKLYLLEKGTRNLTIIETTGLNNLF
SADLLFSQQDSLDIIDHANQFWYAKILQGTEIPSFEPQGVVSGKSDRPYNISYSVGGGNIYEEMVIRHYIEGPTSLDTSG
GTFTTKMYVLSQRTWSYDYSGLDSNSSNFQVGYFTDTRVDSWTDPGDAIRQIQWDGSYQVAGGIKTQFKIGYSLKSKYVP
LTSYEINYQFETPYTAKTLKVFDNSGSNVTRAAGTKYSSGMYLKNTGQTFDVTFTVGHYDTNKQKELYVTWTYDVYNGQS
PYAPANGTKTSNMSFSYYSS
>lcl|NZ_CP026520.1_prot_WP_042235875.1_5319 [locus_tag=PC41400_RS26850] [protein=Ig-like domain-containing protein] [protein_id=WP_042235875.1] [location=5909092..5914764] [gbkey=CDS]
MSNRLRKIWGTTVLIGGILAGWCQTANAAVMMEPLPASADLPSDLAGVSASGVLNGSSLLVTGGQYNRNNSFAKNSYTFS
PSTQTWSTYAPALNLAYHTQSEMKDGRVIVTGGDAFLNGTGFVYNNLTRIYNPYTNAWTQASNLPKNMLGNSQSTLPDGR
IVSVGGADSIFSPSTSALYNNTYIYDPAANTWTEKAPLPIGIYGAAQSTLPDGRILLTGGQSGYVYSFSAYIYDPATDIW
TKAANLPYQYSDLFYRHAQVTLPNGKVLVMGNYNFYLYDPVSNTWIYDTPSVKGMQNAKMTVIGSNVYVIGGYDRNYELN
TTVYKLSFDFTAPTAPVITTSSADWSPADVTATLTAGTDADSGVKQTEYSLSGATVLGWTPYASGDVITVSNSGETTISA
RTIDKSGNASSVTTAVVKVDKTAPTSPVIKGANSAWTNQNASITLTPGTDNGGSGVQRSEYKLTGGTTQDWTPYTGTFPI
STEGITTVHARTIDNVGNVSAESTAVVSIDKASPTIPSVQPAVNGWSKDNVAVTIIPGTDARSGVNRTEYSLSGSTTLGW
TVYTGPVTVSAEGQTTVSARSVDEAGNTSPPVQTVIKIDKTSPSVPAVTPGTTGWTAAASVPVTVQPGTDSGGSGVKSTE
YSLSGAMTKDWTPYTAPFSITVPGQTTVSARTTDQAGNVSGTGAAAVSIDRTPPGAPVIVPAAAGWTSAPEVSVEITPGA
DTGGAGVNRTEYSLSGAETLGWTVYTGRLTVKAEGQTTVEARTVDNAGNVGAVSSSTVKIVRTPPGAPAITTPADKSLLR
SLPAVTGTAEPSSSVVVTIDGKAHPAVKADSGGRFSFSLPAQLADGTHTVTAVAADAAGNTSPSSVAVSFTLDTTAPAEP
VILSPAEGSLSKDTKPVVSGTAEAHSFVYLYVDGALKTIAQADGGGLWSYAIPAALQDGKHNIRAYASDKAGNNSPSTKA
VIWMIDTTPPPAPVVLTPASGTSTNRNQPVISGISEPRALVRIELDGSAAGPVTAEASGKWSFIPAAPLADGVHSLRARA
EDTAGNISPDSDLVPFTVDTRAPDAPVILKPLADALLNTGSPRIEGHAEANAVVRIVLDGGSVQTVTAGPDGSWVYIPAS
ALADGLHAVKAAAADAAGNIGGYSAETAFRIDTTAPAAPRITEPAPGTVTNINRPLIKGTAEASVTVNVYVDGAPAGITT
AGSGGIWSFTPGATLSTGNHRLTALAVDAAGNESPLSQETAFTVVSTNAALHSLSLSGAALNETVTGSTYRYTAQVPYSV
TVTSIQATAEDAQASVRILQNGQPVTNPVVLEVGDQTFTVEITAQDGMTVTPYTVTVTRAASSEVTLSELTVTPGELTPR
FDGAATVYSSTVTNDVYAVTVGAKASSSVATLAINGTVFSGNAGILPLSLHVGSNPVTITVTAQDGVTTQSYHLEVSRAP
SSNVTLSGLTLSEGSLDPAFESGMVHYRAVVGPDTASLTVTGSVYEPNATLVVDGRPVVSGQPSVPLALQPGLNPITVVV
TAQDGITTQSYVVDVTRQLSSNAGLTGLELSAGRLSPAFDGDTSNYALTVDYENDRTTVTASVYDPDALITVNGVPLLSG
KPSAPLALRVGANPIRVTVTAPDRQTVRTYTVLVTRQPRQDSGGDSPGTSPGGTAPTEEPKTPVDPGQPQHPGEPGGTGK
PGEPGVPGKPDVPGGPGCPSPASSPHPVFFTDMKGHWAEQEVTHAADCGIVSGFENGAFRPQEQVTRAQFTVMLVRAYAE
AIRQEQPAPSAGFADSAGIPAWAAEAVALAQGAGIATGYEDGTFRPSAPITRAEMLTLAAKAAKLQPVPAGALRELYADA
DEIPGWAAGYIAAAQQKGLVQGRGPGTFAPLGYTTRAEAVTLLLRMLEAK
>lcl|NZ_CP026520.1_prot_WP_042235877.1_5320 [locus_tag=PC41400_RS26855] [protein=OadG family protein] [protein_id=WP_042235877.1] [location=5915166..5916431] [gbkey=CDS]
MNKNAVLGSTVIIGLLILIGVLGYLLFQKAQGIKAGEKPSASNPPLETQSSAAANGAKQEDQASLVVAFEQALYAAERTA
SDAKKEADREIGQMHQNIMPDSKVILNAYNQFSKVAEKVRKIPVPAGLDPDITTELEKSREKYAQAYDHKAQGMNALMDY
INTTKAGSAVTYGDHIKQADALLLESVGTVIAMQQKLGVDFSKKETPAQKTQTAAQEPASSPSAGSSEASASKDGSASEK
PASPATDKSTSSPAAVAKPAPSPSAVPKPSPVPSSAAPAAQAQEPLSIAGMKKGYFGGIEFGLGTDFKAVEAKWGKVKQD
QHREGGAGYLYKPCNCSFFVGKQNKISAIELNTPMTLQKVIDRLGKPWQEGWDPMEDSYMLVYQFSGNKELYVETQVTTN
NPDAPPPPRSAKTARITLIQK
>lcl|NZ_CP026520.1_prot_WP_042235881.1_5321 [locus_tag=PC41400_RS26860] [protein=hypothetical protein] [protein_id=WP_042235881.1] [location=complement(5916788..5917240)] [gbkey=CDS]
MKRILLIFMCFSLTVLNLIGCSSGKDPVDVGEILFLSTAATIEQDADYNNQFTGVSYSKETLEAISETERKKLFERVAQR
YNASVQQLPEDFIYGRNDLALGIRLSKTTSDKVTVNSTKRMSTGISYTIELVKNEDAWKVEKVTKTSTGK
>lcl|NZ_CP026520.1_prot_WP_042235884.1_5322 [locus_tag=PC41400_RS26865] [protein=endonuclease/exonuclease/phosphatase family protein] [protein_id=WP_042235884.1] [location=5917496..5918296] [gbkey=CDS]
MTNSKNLTAMTFNLRVNVLSDGSNAWPHRKHLAADSILGSGAHLIGTQEGTAGMLADLDAQLPAYSRIGNGRLGEAEAKL
HQDEYCAIYYNREALTLIKQGQFWLSETPEEQGSLGWDSSYPRICTWGCFESKEHPGSRFYAFNTHFDHLGQKAREESAK
LLLERIRACREEENAPVLLMGDLNAEPDNPAIGLLREQLQDAYMLLDEPVGRTFHDFKGGTDGEPIDYIFATQDIEMIRT
TVDREQLDGAYPSDHYPVELTWGFKG
>lcl|NZ_CP026520.1_prot_WP_042235887.1_5323 [locus_tag=PC41400_RS26870] [protein=FG-GAP-like repeat-containing protein] [protein_id=WP_042235887.1] [location=complement(5918525..5921734)] [gbkey=CDS]
MVRSVKSSPVTKWLALVLGASLVAVPVGPVKANGANPVFENAVAYEVGGWAYDVTTGDVNGDGLDDLVSLNFQKNQVAVL
LGRGDGTFGAPASFAVGSNPRAVVIADLDGDHHPDLAVANFMDNTVSILKGNGDGTFLTGGSVAGVGPEATGIAAADLNG
DGFPDLAVVNNVSGSVSILFNIGGSGNFVLGASIPVGAYPRSLAAWDMDGDGKTDLAVANSGSNTVSILKNDGTGTQFSR
ADYAAGKSPNRMKAADLNGDGLKDLVFAANGDNGVAVLLGTAGGGFGPSVLYPSGNLPLDVALGDFDGDGHTDILSQTQT
QDFAILMGNGDGTFQPPITYPYPRPAIYLETGEFNGDGRSDIVLAGASSSNTSVYVINSKAPGAFSFDSDTSESAENGGT
ASVTIKRTGSMAGQTAVRIRTTDGTAVAGKDYVPVDEVVVLQNGEASRVYPITVMDNQIYEGDKTFTVKLSDPANGYLIG
APNEAVVTIREDDAVPDTTPPVVDISKFGSTDNYTGTPDRLYGAAGAVGEAGAVVKAYPWTDTDKDGIVSAGELGTPIAL
GSSAADGSVPAADIGDLAPGSYAYVITATDAAGNESAKGAWAVLTVTLAKGASPDTEAPVWPAGAALVFSDVTSGSVNVG
WPEAADNRAVTGYEVYRNGALLTSVSADTYSYRAAGLTGDMLYSFSVKAKDEAGNGATELAGTVRTLPVPSSGGSSGSSP
SLSAESRLQELRFMAGEKEAVLTPSFQPGVSDYAAETDADRGVIRFKTMDSVARAGVNDGTVNTSTGEYAAALNEGVNVF
KIAVQAEDGSIREYKLTVTRKAAGGGTGPGTKPDSGPGTGTGTTPVPGTKPEPGAEPGTGGGSRFADISGHWSEADIGRA
VALGLVDGYEDGAFHPDATINRAEWIVMLERLLKNADTAGKTSTDAASKTSAGSGAETAGTVAANAAPLPFADAGAIPAW
AVRAAADAAANGIVSGYDDGTFRPGAAVTRAELAVMLAKAVNGSAVGGGKAAAFADEADIPAWARPYVQTSAERGLIDGR
SGNRFAPNEEATRAEAAVLLVRLAMLLSS
>lcl|NZ_CP026520.1_prot_WP_042235888.1_5324 [locus_tag=PC41400_RS26875] [protein=polysaccharide deacetylase family protein] [protein_id=WP_042235888.1] [location=5922087..5923172] [gbkey=CDS]
MTNIVITRGNPAVSHCAFTFDDGPMRIPIDAWLDALEQGDARGTFFLTGEWFDRYPAKAREMLARGHELTTHTYHHRRMA
DVTKSVFFEELKLAELAYQEATGRPVPAFMRFPYASYREENLEWLREWNYLVVEGEDTVDWSGPPSAELVERVIPKLVNG
SIFMFHANEIAKETPQAVKSLVYHAHAKGLAVVPVSELLHADGIRAGERIWQVRFKPTFQGSFHSEQWKRVAGEDELRNL
AADSLEWGNPKAPTGSGAYSKWLQMLSKAFQSDGQTRFVARSFAGQHWAYVHASVRGGTLALEDFATKEAHADALVSILQ
WAAHEASALGCEWITSTQDMRRIHKMCEQLGFEAEIVLQEG
>lcl|NZ_CP026520.1_prot_WP_129112541.1_5325 [locus_tag=PC41400_RS26880] [protein=non-ribosomal peptide synthetase] [protein_id=WP_129112541.1] [location=complement(5923496..5943151)] [gbkey=CDS]
MEQYLTLIDIFQDRRGQAEKGITYISGDRDELYESYGELFEHALQALHYLQSRGMEPGDELLMQIDENRTFLHVFWGCLL
GGIVPVPVTVGSNDEHKMKLFKIWDTLRRPYLISDAKVLGSLEKYARQQGLEQAFGVLAQKALPTESLDYGSEAKGIIYP
VEPDRLAFIQFSSGSTGDPKGVMLTHENLVHNIRDAAAESELDDNDAYLGWMPLTHDLGLIAFHLTCVIGNVNQFIMPTA
LFIRRPSLWLKKASEHRVTLICSPNFGYKFFLDQYKPEAAKDWDLSRIRIIYNGAEPISTEVCDRFLSAMAVHGLRRSAM
FCVYGMAEACVGVAHPPLNEGIKAVHLNREHLTIGEAVVPVAPEERCCVTFVDVGYPMASCDFRICDDNHLPLPDDVIGH
IQICGKNVTSGYYNNPQATAKAMTKDGWLVTGDLGFMRSGRLIITGRAKDIIFVNGQNVYPHDMERVAEEVDGVELGKVA
VCGVRDPGTGQDRIVLFVIYTKKPENFVPLIVKLRHHLNERGGWEVRDVVPVRRLPKTTSGKVQRYKLAQQYMAGEFDEV
SGRLAGLLLAARAAKPKALARSEAEQKLLALCKRVLQTEEIGTDESYFELGANSLQLTQLAEALEGEFGKPIPVTDFFSY
PTIAKLAAHLAADPAQKAPVKLEAGPMDDPGDRDIAIIGMAGKFPQADTLEQFWEHAAQGYDAIGPYGDARKRDADVFMA
KLQREPGQWEAVEGGYLDEIDTFDYAFFKLTPREASLMDPNQRLFLQTAWSALEDAGYGGAKMAGARTGVYVGFSKTSFD
YERLLTEAEPEALPRFIIGNLPSVISSRIAYLLDLKGPAVTVDTACSSSLSAVHLACRALMNGDCDMAVAGGVKTILLPI
RAGIGMESADGRARAFDRESDGTGAGEGVGAVILKPLRRAVQDGDRIVAVIKGSAMNQDGSTIGISAPNALAQADVIAQA
WRNARVDPETITYIEAHGTGTQLGDPVEIEGISQAFGRYTERKQFCAVSTVKTNIGHLYEAAGIAGLIKAVLSLRHRQIA
PLVHFRAPNRQIRFEDSPVYPNVSLTDWEPQGFPRRCGVSSFGFSGTNCHVVLEEYKPREEAAEAGAGGTAPDGALLLVL
SARTDSALKALIGRYAERFERSPDLDMHSVCFTASTGRSTFVHRLAIAAANPGALRDSLLRLHREGIADEGVFSGQIGTV
SGYGEPAREVPGGPEAAGPFSAGEGGAAPTLETAEEAARLAEAFVRGEPIDWEQLYAQGQRQRVSLPSYPFEGKRCWIQI
AETPQRRREETGSLTMEEGQVNGVSTLHEQLSHAGAAADSARHSEVAQTLAQMISDVSHLEMNELHPRTHFLELGLDSIN
LTQVRQGIKDVFGLDIPMSRFFENLTSLEQVAEYIVHHLPDNASSGTQNGIDRSGRPTGDAVPHRADVLSAAAGQSGTTE
GLPAFLPISAAPRQDGQAAHDIHAGLAAVSSLGQAAAAANTRSGRTPADAERSGGAEAQPGWNGPPSAEALRQTAPEPTG
GWHHPPSVAFVHALRPIMEQQLHIIARQLDVLQPPSFVAPESPEYAAGTEPPAFPAAVPHAHSVDARSEAAAAGGSPYVA
APSTVSGGDHGTDGDSSEPARAATADSEGGQPFRPYRPIDVKARETLPLRQEQHLRDLIDRYTARTGQTKAYTQKYRAVY
ANNRNVAGFRLVLKEMVYQLVAERAEGSKIWDLDGNEYIDLTMGFGVNLFGHDPAFIREHIDKALRSGMCLGPMSNLAGQ
VAEGICRMTGVERIALFNSGTEAVMVALRLARAATGRSKVVLFAGSYHGTFDGVLALARPGGDRERSMPLAPGVPAGMVD
DIVVLPYGTETALAYIREHAHELAAVLVEPVQSRRPDLQPRPFLAAVREATEQAGAALIFDEVITGFRLHPGGAQAWFGI
EADLVTYGKVIGGGLPIGIVGGRAAFMDGIDGGMWDFGDGSYPQNEERRTFVAGTFCHHPLAMAASLAVLERLERDGEKL
QVELNARTADLAETLNRFFTEEKVPIRIDYFGSLFRFVLKGDLELFYYHMLDQGIYIWEGRNCFLSTAHTETDIACIVQA
VKNSVAALRHGGFLPDSPPDGPDPGKLPLPERPVHDQDIVIPLTPEQKQLWFASASSRENPHVLHESALIRLRGPLHLQA
WSESVHTLMHRHEALRTYMGVDGETQVIASSLKAQIKLLDVSGDDDLSRERQKSAWLKQGIETPFRLSESEPLFRIRFLK
LEEEEHLALFTFHHLIADGWSIVVFMRELASIYSALVRKERYRLPEAAVFQDYAAWQRLRMAESRSAEASAFWTGELSGP
LSVLELPSPLRGVTVPGLEGGRYTAVLEGGLVQSLKTLSIRLGSSLFVTLLSGFKILLHRLTGDSRLTVGVPTAGQAQMD
AFSLMGHCVNVLPVVSRIQHETTFEDFAGAVKESMRRLEAYQAYSFAALAEQGLRHLPVINVVFNMDRPLPKLRFHELDT
ELLPSPVTASKYELFVNVTEANKELRVDFDYNAGLFEPDVIAHWFRYWVKILNEITKRSSQPLAEISLLDMDETRQIYKH
WSSLADRDGGCPCVLDAFGRTAPAGTAGELHIAAQGLLNPQSMREAARTGEWVYHAPGGELKRIGPISAFARIRGRLINL
RQLETHLLRVCNLEACVIRVWEDSAAGGAATLTAYVVPDPSAPWEEEALKRRWSEALPDYAQPGQVMLMKALPLLADGEP
DWERLPAPSMPGADLMPEMGEGPDDGVEAELIRMWQEVLGVSTVRAHDPFFQLGGDSLTATVLLSRIRKTFGVQIPLGEL
FHRQTVDALAQYIRGGEKQAFSPIEPVEAHPHYPASPGQRRMYLLQQQEGGTTYNVSGCLHLEGDLRVPSFIEALHEAAG
RHESLRTCLKLEGGEVVQVIQPELVLHVPVTQAEASELEAISRRFIRPFRLEEAPLIRAELLDFGNRRYALLVDVHHAAA
DGFSMAALLDEVLQAYQGRQLKPVRVQYKDFAAWQRRQLAQGTYREQEAYWLQTLGGELPVLDMPADFSRTQTQSFEGGA
VSLTLTPQLTDALRQLAANSGTTLYMVLLAAYFALLSQYTGQDEIIVGTPVSGRRHADTETAVGMFVNTLALRGRLQASQ
SFGELLQHVKALALAAFERQDYPFDELLDKLNPVRDLSRNPVFDTMFMFQNEAVQPSRAGALTYKVEEINPGISKFDFSL
QVTKDPTDGETLQCTWEYAVRLFVPETIRLLAKRYIRILEVVSADPAVRLTGIDLLTDAEKERLLDAGNGADCPAVPEFT
VQGLFEKQAFATPDRAAVVFDGQTVSYGELNRQANRLAHKLQALGAGPNTLIGICLDRSPSLPAALLAVLKSGGAYVPLD
PSLPDERLRYMTERAELRVLVTVQAYAERFGGLRDLSVVCLDTEAGALLSQSDANPRSEADGEHLMYVIYTSGSTGMPKG
ASVYRSGFASLMRWYTGEFGMSAADKLLVITSPSFDLTQKNIFAPLLTGGQLILPPAGPYDARQIVSLIERHGITLLNGT
PSAFYPLLDVSAPAQFAPLATLRHLFLGGEPIAADRLAPWLKSDACLAEVVNTYGPTECTDVTVFARLPAITALAGKRVP
IGRPVPGTRVYILNAELGLVPAGTPGELCIAGAQVGGGYVGDPQQTAAKFVPNPYGGEQCPVIYRTGDLARFLPDGTIDY
LGRIDHQVKVRGYRIEPGEIEAVLRHDIGAADAVVLAREVRPGDRQLIAYAVPRVSSKEAAMPPKALAALWRSELRRRLP
EYMVPWPLVIMDAFPLTPNGKIDRSALPAPDVDLAERSGEGTEAPRTVMEARLAELWREVLGAEAVGIRDHFFEIGGHSL
RAAALTARIHQALQVEVPMRMVFEHPTVESLAQAMEELQPNPYAGIPAAGEQESYPVSSAQKRLYVLSQLETDGFGYNMP
GVLHLEGEPDAARLEQAFRALIRRHEALRTAFTLHDGVPMQRIAPKEAPPEFTLPRIRVQGEEEALQAARAFIRPFDLEQ
APLLRAALVDEGANRHRLLVDMHHIVSDGVSTAILLEELSRLYNGEGEGEALPELRIQYKDYAVWQQSEAGSERVRAQEA
YWLERLKGELPMLDLPADRTRPAVFRYTGGVVRFTLGAERTTGLKRMALRTGATLYMVLLAAYSTLLSKYSGQEEVIVGT
PVAGRPHAELERVMGMFVNTLALRTYPSADKPFEAYVREVKELALEAYAHQDYPFEELVEKLNVRRDMSRNPLFDTMFAW
QGEGGAVPELAGLKASADEADYRSAKFDLTLDAEEQGDELLFRLEYCDVLYERKTAERMAGHFVQLIDAALADPQAKLSA
LSMLTPEERTQVLESFNDTTRAYSPDDSMTIHGLLERQAARTPDRPAVVSGKRQWSYRELNERANRLARTLQAKGVGRES
LVGILAERSPEMIAAALAVMKAGGAYVPIDPDYPEARIRYMLEDAGVSLLLAQSPLRNRVAFGGEWLLLDDPQSFSGDGS
NLEGSVEPGDLAYVIYTSGTTGQPKGVLVEHAGVCNYKLFYDEALKVCEQDRVLLFASFSFDAACSEMTMSLFGGAALYV
PDASVIADYRLLEQYVRDNGITVATLPPTYAAYLNPVHMPSLTRLITAGSASSPVLARRWSGFVRYFNNYGPTEDSICST
AWPYSTLDDTAKTVPIGRPIANHQVYILGADQSIMPVGIPGELCVSGIGLARGYLNRPELTAEKFVPVPFAPERRMYRTG
DLARWLPDGNLEYMGRIDDQVKIRGYRIELGDVLTQLNRLPSVREALIVAHADKAGAVELCAYFTAEKTWSTGELRQALL
QELPAYMVPTYAVQLAAFPLTPNGKIDRSALPAPDVDLAERSGGGTEAPRTVMEARLAELWREVLGAEAVGIRDHFFEIG
GHSLRAAALTARIHQALQVEVPMRMVFEHPTVESLAQAMEELQPNPYAGIPAAGEQESYPVSSAQKRLYVLSQLETDGFG
YNMPGVLHLEGEPDAARLEQAFRALIRRHEALRTAFTLHDGVPMQRIAPKEAPPEFTLPRIRVQGEEEALQAARAFIRPF
DLEQAPLLRAALVDEGANRHRLLVDMHHIVSDGVSTAILLEELSRLYNGEGEGEALPELRIQYKDYAVWQQSEAGSERVR
AQEAYWLERLKGELPMLDLPADRTRPAVFRYTGGVVRFTLGAERTTGLKRMALRTGATLYMVLLAAYSTLLSKYSGQEEV
IVGTPVAGRPHAELERVMGMFVNTLALRTYPSADKPFEAYVREVKELALEAYAHQDYPFEELVEKLNVRRDMSRNPLFDT
MFAWQGEGGAVPELAGLKASADEADYRSAKFDLTLDAEEQGDELVFRLEYCDALYERKTAERMAGHFVQLIDAALADPQA
KLSALSMLTPEERSGLLDHFANAGLEAGRGMAGVFHEAFTAQAARIPEQTAVVYRDERLSYRELDERSNRLARKLQACGV
GRETLVGILAGRSPLLLVAVLAVWKAGGAYVPLDPDYPPERIRFMLEDSGASVLLTEAALEPRLEVILAGNEAGTAVLAL
DDEAIYDDHGSDVAVINRPGDLAYLIYTSGTTGRPKGVMIEHQSLMNTAEGYRREYRLDQFPVRLLQLASFSFDVFVGDL
ARALYNGGMMVICPKDDRLDPNRLHHWIQAYEVNVFESTPALIVPFMQYVADRGLDLGAMKLLITSSDSCNVADYRALQE
RFGAQIRIINAYGVTEAAIDSSYYDQPLKRLPDAGHVPIGRPLLNARFYILDSQLRPVPVGVLGELCIGGAGVARGYWNL
PDLTGEKFVPNPFVPGERLYRTGDSARWLPDGDVDFIGRLDDQVKIRGHRIELGEVEARLMKQSSILEAVVAGKESIHGQ
KQLCAYIVANGVIFPGALRKELAEELPDYMIPSAWVQLDRLPLTPNGKVDRKALPAPNREENPDAAYAAPHTKLQRTLTE
IWQKLLGAEKIGIDDNFFERGGHSLTAIQLIAQAQAAGIPIGIHHIFQGQTIRKLSALLEQPAEVNLQPAETVQESERRL
KQAFGLRSRFVTEESEDGLRVTLNMEPCHGVKVSELLAFIRAELAPNLHPHYLTTGESADDSFAEGRDAAIEQLVPLAIS
EKSPAEIKRRVKKELSRIQVLHEALQSDINAGAVECRYALAPAQEYHLERPQCSGTVIRLDVLGLEADRLSEAVRSLLAR
HSLLRSVLVRGEQDWMWEQRAVPAQLTLPAVDLSDCSKADQRELLGRLLPKLFTEPYRLSDRLLYRMALVRLNWREHWLL
LPCSHLIYDETSGSILESELLEAYRRLDSRSEEDLTAWPQLRTPGAETGTAGMPADGYPAYVAQVRKGPQHIGDDELVQL
FQLEDFDAGLELVNESIRAPLGQGYTKAELRWDMSEFEELASVNWGVWSLQLAGRFFRSYLQLAAVPVWMTHYGRHYENN
AFFKTVGECIDQIPLLLTGTEADSESVREKIQTAAARNINFLNLVYNEQTASNYPKSRHLLRKGLDSIAIVYNHLGESED
AAYLFEEAEAAGGESQAQQSPVLHFTSQHQGSVWQISVTLPYREEKETIMEVLEKELQALVASLHSVKEPV
>lcl|NZ_CP026520.1_prot_WP_042234711.1_5326 [locus_tag=PC41400_RS26885] [protein=VOC family protein] [protein_id=WP_042234711.1] [location=complement(5943725..5944159)] [gbkey=CDS]
MDKATPFLWIKNTAEGIEFYKKAFDAVEKYRLTEPGGKVAHAEITIGGAKISIAGEYPEYGILGPETIGNTTVGIQLQVD
DADKLFQQAIAAGATMVNPMEDQFYGDRAGSVKDPFGHYWMISTTIKTVDPEEMQKRFLALYGS
>lcl|NZ_CP026520.1_prot_WP_042234713.1_5327 [locus_tag=PC41400_RS26890] [protein=helix-turn-helix domain-containing protein] [protein_id=WP_042234713.1] [location=complement(5944346..5945167)] [gbkey=CDS]
MLLRTYIPQAPLSVFVDFFWYFECYSPPHSRELTLPDGSIELVFNLSEDRTKWIDQDYNGLFQKYGSSVISGPHSRSFII
DTPVESAILGVHFKPGGIYPFLKMPVYELHNIHVSLDTLWGARAHDLHEVLLLSKTVDDKFRKLEESLIKLASRPLIRHP
VVSFALNEFLGLPHKSPLSDVIGQIGLSQRRFIQIFKEEVGLTPKLFCRLRRFQEVLSRIEHAKSVNWLDVAIACGYYDQ
MHFIKDFQAFSSLNPTDYFSRQGRHHNHVAIHN
>lcl|NZ_CP026520.1_prot_WP_129112542.1_5328 [locus_tag=PC41400_RS26895] [protein=hypothetical protein] [protein_id=WP_129112542.1] [location=complement(5945361..5945570)] [gbkey=CDS]
MTLWMICFQKKTIQPKTVVENIDIDIAGRKDSFRSFEWDTSYEEVKAESCQLKESDDTCGVLNMDSSKK
>lcl|NZ_CP026520.1_prot_WP_042234718.1_5329 [locus_tag=PC41400_RS26900] [protein=LysR family transcriptional regulator] [protein_id=WP_042234718.1] [location=complement(5945593..5946495)] [gbkey=CDS]
MDLRALKTFEACVRLGHFLKAAEELQYAPSTVTLQIQRLEADLGVSLFVRDGKRVIVTEAGRWLHNEASALLKSIGTMRQ
TVSDIAAGDTGSVRFAAIEPVASQQLATVIADFCKTRPKVELTMEVSGSRTIAERVRAGELEFGVCSAPSSKLMLEFEPL
IEERLGIMLNRNHPLADQDSITASDLAGLTVLVKEPTCIYRELWETAIDGAGRNPFSCMEVGSFFVIQQMVKAEFGIGIV
PIYSGLEQEGSLVIRPIADLNPVVTIGIAYKDKNYLGKAALLLMEAIRGIREPNPTRPSA
>lcl|NZ_CP026520.1_prot_WP_042234721.1_5330 [locus_tag=PC41400_RS26905] [protein=YbfB/YjiJ family MFS transporter] [protein_id=WP_042234721.1] [location=5946612..5947850] [gbkey=CDS]
MGTKSAFNRITLWASFLFVVILGFSRLSYGMFLPGIQRVIGGSYGQLGILGTVNFIGYLVGTLCLPPLISRFPDRKHVIN
RITCLLLGLTLIGSAFSDHFTGLGLWRFAIGLLSAFATVLVLSIAMDAVRPAERGAASGLIWLGGSAGILVTGLFAPFTI
DPSHLQGWRYAWVIMGIFGVIAAFGFEFVTRKREAEKMPSQTESKSRDKENENVYRLLLNPKKLLFLIASYFFFGWGYII
YFTYLIPYLVSKGIPSLYAGLIWSGIGFAGLFNGWIGGKAIDRWPSGYTLASGLTLGTIGVCGVITNSMLFTVLGAAMIG
LVSFITPPLMTTALLRRHVPHHAYAACLSIATAFFAAGQIIGPIVGGVAVERYGLQFGVASSAIFMAIAAAFAGLYGQQQ
RKLEIRDLPDSG
>lcl|NZ_CP026520.1_prot_WP_042234722.1_5331 [locus_tag=PC41400_RS26910] [protein=DinB family protein] [protein_id=WP_042234722.1] [location=5948009..5948506] [gbkey=CDS]
MYATVQDLIVEWKNEAVLTQKVLDGLTDHALGQQVYPGGRTLGRIVWHFVTNIPDYLTEFGVLVEKVPNPNEVPSAKEIA
ETFNKISSRVVQALQEQWTDDSLKHVQNAFGRMESNATIFMGLIKHIVHHRGQVTVLMRQAELPIPAVYGPSKEGWVQLG
VTPPL
>lcl|NZ_CP026520.1_prot_WP_042234726.1_5332 [locus_tag=PC41400_RS26915] [protein=ATP-binding cassette domain-containing protein] [protein_id=WP_042234726.1] [location=complement(5948791..5949462)] [gbkey=CDS]
MAQVLLTDIAKRFKDKTIFEHIDLTVAKGRTIAVKGRSGSGKSTLLNIIAGLEKPTSGEYFLDGEDMSQLNLNRLSEIRK
HKIGYISQHSPVIPKLTALENIMVPFWFEKKRENSKIIQKQVERLGRLFEIHHLLNDRVEKLSGGEIQRVGIIRALIKDP
ELIVADEPTGALDDETALKILACLDELKVNGVSVIIATHSGMVAEKCDERYQLTNQGLIKEVQ
>lcl|NZ_CP026520.1_prot_WP_241688922.1_5333 [locus_tag=PC41400_RS26920] [protein=ABC transporter ATP-binding protein] [protein_id=WP_241688922.1] [location=complement(5949469..5950731)] [gbkey=CDS]
MILLFIKQNKLLNMTIFLISLIFCSLLLVLTLNINQTRIEIEATRNFKDKNIYQISDNLYKEKEKQFFSKPDSFEILNNF
SNQVHANESYKYYNAVWQPIEISDFKGDIIYDAYYETGNNQPSYMIQNKSYRAIKSVQMNQAVFDFNHLQLETGVGFKAD
DYVFKQEDDRLPIILGSEYSSVYKTGDTLDILYYQRKFTAVVVGILEPSQKIMTVREPELLLNRYIILPLLDFEQSPPKM
EGIFLKALLLSKINGNIITDLEPLQIRTMLNEISAKTGFKEFSVIGANSLAVNSIVNMTEINRNTLFVISVALFITLIAL
FIIIMRIIIKKNIDFYKVLLISGVSMNQIYRSINYQLPLLNLIGCMLPLVFIMVTLNNPLSLLVNYLIVVGISSLIIAIA
VNRYTRKYFEKVDIVQQLKG
>lcl|NZ_CP026520.1_prot_WP_241688744.1_5334 [locus_tag=PC41400_RS26925] [protein=hypothetical protein] [protein_id=WP_241688744.1] [location=complement(5950734..5951564)] [gbkey=CDS]
MLNNGYKDQKAIFFKMEALDVPVHLNAGKFILFQYDDDSPALKHVWVNGELKLPPIKRTDEGAETDKIAVVGTNAEVKNI
PNDYKLIGYFNTPYSYKLNSEIWLLSKNPQIELMDGKQFVFISPSRDVKSVFDQFINGNNVEIINAETNGSYILKSNQVV
GMISYITILFMTTIFCVSTMIWIKRKNHLLSVLYIFGYSYRAIYRIILRYYALPYILFSSILFVIFMLISYYAVPLWDVS
WMLYSVNLILLHLAIVIIFVFYLTFIYTVKKGGKRF
>lcl|NZ_CP026520.1_prot_WP_241688745.1_5335 [locus_tag=PC41400_RS26930] [protein=hypothetical protein] [protein_id=WP_241688745.1] [location=complement(5951735..5952160)] [gbkey=CDS]
MNTKKMVPLALSVGIILGSVAPSALHAQSTDANGAIGGWSEKDGYFSITAVSSFIGIASTAKHTGYVEDIDRGNVLFSRA
VGNTSWPGVYHYTRARFEGHITGNILGDSGRVWGTNATEAKSDWVDRYASDWVGKTYYGNE
>lcl|NZ_CP026520.1_prot_WP_042234733.1_5336 [locus_tag=PC41400_RS26935] [protein=restriction endonuclease-like protein] [protein_id=WP_042234733.1] [location=complement(5952387..5954804)] [gbkey=CDS]
MALPPSGSLNQGTVLLRVETNWFNFYIRGKPFHPTVESLQLHREGQGVNAHLLVSALGSSVTVEKASVFDPETGELAEWQ
PGEWRNPLFYETQAYQVVVEKKEDLALTFHHENVHLRQAVEPLGQMILAGELNFQNEVGLTELQLRLHGEPIFQIQLEIF
PAKMDYQNDYRQILHDVNQQIYNLSFDFLRKTYHMTGLRETGSQSLTEFFTILQVVFGQLVQAVERIQASPHASMRVENR
VVEAARVRKAGRETAAYLTKRPHLLVEDKKQGFISMNGRMYTPRHALEARRQVDYNTQENRFVRWVLVRVSKKLSELKTR
LTQGGRMQDPLLAKRIQGMQGQLQRLLGLDFLQVGEMRQLTVTLVLQMAPGYREVHRHYLMLMKGLSVQSDLFKLSMKDV
AQLYEYWCFLKIHELLRRKYELVSQDLVRVNRTGLFVTLDRTQKAKMSYRNPVSGEVFSLYYNALPPRDHSKTLGQKPDN
VLTLKKNDSPVEYKYIFDAKYRMNPAVEGTPYYNIYKSPGPVEDDINTMHRYRDAIVYAEGSGHELEREYERSMFGAYVL
FPYGDEEKFREHKFYRSIGLVNVGAFPFLPNATSLMEAFLDEIILDSPEKAYERSIRPRGTKDYYRNKLEGKNVVVGSMR
SGPVQLEAALKHGFYHLPLKNLTEQKILTQLEYVALYQSIKGFGKNGEVGVHWYGRISDWKVVRRREITEIPYRPGTADD
LYVKLAVEEWKRMERPVMPGGRGIHTLLLTSKYMLDRAVEIAELKLESEADLAQWREKRRQGSVKVVLDHQEVDLARQVL
GVEVR
>lcl|NZ_CP026520.1_prot_WP_042234735.1_5337 [locus_tag=PC41400_RS26940] [protein=DUF3578 domain-containing protein] [protein_id=WP_042234735.1] [location=complement(5954776..5956995)] [gbkey=CDS]
MTLLSELSSIFRHKQKSYKMVLILSMLEEYRNSQQQLLSFGQVAERFLSYYRMNEDTGNKVDIPPGGSAARWSTFSLTQT
KSLLNTPLNALSAILDFNNDQQTITFKESIWNQLNEQELAELKQYAEQELEQYNLQLRTVFSLRQALEDTFNTYTSAKNE
PFANHALGTLFRQAIPEYFRSQPFINENYKVQGSIGQGNWATIPWIAVLDRRITETTQHGEYVVYLFSEDMSSVYLTLNQ
GVTVPLRERGRRDGHHYLMQKVQEMRELLPLNGMQKDDNILLTTSGLGRDYQVSTVAYIRYDKDHLPEEDQLLADLENVM
NNYKLYVDQVVGQTAEGDEELVEEIERPIPTDTLTVLERIQQIKAYIEQEGFHYPGALIENLYLSFKTKPFVILAGVSGT
GKTKLVKLFAEAIGATSENDQFTLIPVRPDWSDPSDLLGYKDLSGVFRPGRLAEVLVEASTAENRHKPYFVCLDEMNLAR
VEHYFSDLLSIIETQKWQEERIVTSPLIHAKALNPEDRSIYGGLALPDNVYLVGTVNMDETTHPFSKKVLDRANTIEFNY
INLMQYPGYEEPSEGTAVVPASNGFLRADYLQLVDAYPNYLELIQRTTEKLVTINEILEDIHSHVGFRIRDAVCFYMVYN
QRFELLSENEAFDWQLLQKILPRVQGSSSSVKRALLGLMQVAVGKPLNMGALMDDASELTRKWGSRQSTEVAPYPQSARK
IAFMIRRLEEDGFTSFWLS
>lcl|NZ_CP026520.1_prot_WP_042234737.1_5338 [locus_tag=PC41400_RS26945] [protein=hypothetical protein] [protein_id=WP_042234737.1] [location=complement(5957023..5957820)] [gbkey=CDS]
MWKDIAALFGSNPLVTLLIVVVGTSTLWMYKEFKEILNRNNIAKLNLINERLKTFGKLEAAIALALKKPEDENILVKLYD
ILGESSALFTKEMREVTRTFYTQGHPHILSALQIFIDNQINTSREEKAELSKYENSTDVIDKIGRMIKPFVPIVFIWAFV
LLLVSYVSVLIHQQDMSAKIHWTMYFFSVLISFMFVCVLLSVDFDNKLGKQGHYRWLLLSTIMITPFVFLLYTGLWWVSI
SIQVIAFILLIKKQKKAKNSLILLK
>lcl|NZ_CP026520.1_prot_WP_042234739.1_5339 [locus_tag=PC41400_RS26950] [protein=restriction endonuclease] [protein_id=WP_042234739.1] [location=complement(5957866..5958852)] [gbkey=CDS]
MTKWWMVRAGDNNELIPEWQANKIASIGWPQLGDPRKFASREKFVEHAHSVYVDQKPGTRLSWAGQVWRYYNEVALNDRV
ITYSRETREYLLGTVTKAPEYRTDINALYPNVISVAWESKRIPRDLFSQRAKYSLGSTLTVFRIDDWQEEILALFHQTPA
APVNVTVQDEEAAEFNTEEFEGRAKSLIEDRVDKLDPWEMQELVAGLLQAMGYQTKVSPKGPDGGVDVLAHRDAFGFEKP
IIKVQVKHRRATSGGPEVQQLVGASPHGASCLFVSTGGFTAAARTAAQHNGVKLLDLAGIVELINEWYDKMPAEKQALIP
LKRIYVPM
>lcl|NZ_CP026520.1_prot_WP_042234741.1_5340 [locus_tag=PC41400_RS26955] [protein=helix-turn-helix transcriptional regulator] [protein_id=WP_042234741.1] [location=5959259..5959486] [gbkey=CDS]
MSKKVVIKIGSLTKKRGISLRELSRLSDVRHAALSELSNGKRTNINFSHIERIADALGISDIREIIDLVETTDKD
>lcl|NZ_CP026520.1_prot_WP_042234743.1_5341 [locus_tag=PC41400_RS26960] [protein=hypothetical protein] [protein_id=WP_042234743.1] [location=complement(5959496..5959768)] [gbkey=CDS]
MSSLHWRNNSLIHRFDKLADEADTSMVFQPMTEQLSILEIELMQSLEPDQQILFREWQDKELALEYMKKEWIYVRGVQDG
IKILSQLYSE
>lcl|NZ_CP026520.1_prot_WP_042234746.1_5342 [locus_tag=PC41400_RS26965] [protein=helix-turn-helix transcriptional regulator] [protein_id=WP_042234746.1] [location=5959876..5960100] [gbkey=CDS]
MSKKIVIKIPELIKKRGISLRELSRLSDIRHAALSELSNGKRTNINFSHIERIADALGISDIREIIDLDYIDDK
>lcl|NZ_CP026520.1_prot_WP_042234748.1_5343 [locus_tag=PC41400_RS26970] [protein=DUF2075 domain-containing protein] [protein_id=WP_042234748.1] [location=complement(5960145..5962013)] [gbkey=CDS]
MIIYSSNALKFRESVDNNQITKQIESAFVKRMGKKPSQGEKRAWNNSMQFMERIIRNSHIADDCGILIEYNIPSTSKRID
FIVAGKDDNDNDNFIIVELKQWDSAQATDREDVVKAFIGGNTRETPHPSYQAWSYRQFLNDMNEAIFTNEINGYSCAYLH
NYPTINPDPLQSSQYEDIIREAPLFLANDSIKLQEFLRKHVGKGNGLNLLYLIENGRIRPSKKLIDHVDGLFAGNSEFVL
LDEQKVAYETIMSLAKETETKKTIIIQGGPGTGKSVISMNALGGLLKHKLNAKFVAPNASFRTVMVETLTKQKARNKTRT
KMLFAGSGQFYDTPENFYDVLIVDEAHRLKGKGAFQYRGDNQIEDIIKSSKVNVFFIDDFQRIRPDDIGSIAEIRRIAEE
NHSEVHKYSLSAQFRCSGAEGFINWVDDVFQIRPTANYDGWDHRSFEFDIVDSPNKLYEIIKSKNEAGFKARLLAGFAWN
WTAEADGNGNGEVTDVIIEEHNFQMPWNARSISNTWAIHENGINQIGCVHTSQGLEFDYVGVIIGNDLKFDLEQLKLYAD
YDEYKDRVGKRGLKYDNEKLSILVKNIYKVLISRGIKGCFLYCRNKELQEYLCQRLNIVGQR
>lcl|NZ_CP026520.1_prot_WP_042234751.1_5344 [locus_tag=PC41400_RS26975] [protein=hypothetical protein] [protein_id=WP_042234751.1] [location=complement(5962104..5963204)] [gbkey=CDS]
MLLGSEYEWLSRIAGVYNRHITTGQFQFTENDYKLKFSYDSSLLELEKLIYKGYVIRRDNLIILDSIQWINTFLDGINKS
LIKYNNKLKCEYDLQENKVFIYENNEKIKGYSLNFSKMATPEDEVINFVRLASFDFYIYWLDLLNDNELFEMFYYHLTNK
LSKFKYDKRIIFDFLDDLETSEFNEIYETTLKEYFKNKEMDIRELTNDQGQVLKKFLLSDIFNAYSVSIHSCIIFVVKKD
KKIEFISYETSEIYDESVIGEIEVLRNKLLKKVTLYRSLNRFNKNKVGENTQKTIQIASLLITPINILLLFGISKNISWI
KDLVNNYYLYIVIVGVLIISLIGAFLWVIIPSIRLSIFKWKLPMKD
>lcl|NZ_CP026520.1_prot_WP_042234753.1_5345 [locus_tag=PC41400_RS26980] [protein=hypothetical protein] [protein_id=WP_042234753.1] [location=complement(5963188..5964582)] [gbkey=CDS]
MSMKERKLFFTRNGEYTVWEMLNALTDTYLEELYEFALTKRWTELYPKSNDEFPSFLFNIFEELDELNNNNFLTQIQDRF
YMVPPPRSDFNAIVFHYYGTSLDLGSVEVKLDELKRNLVTFGIDGLQLEFFIYSEINGHKNVVDLRFTSEKSSYYKKNEN
IQFLNTEIRIYLNSKIALLTNFSQYTHSDKDKYNFINNIIRNVSSYSGTDLKPIHLSDQSLRELLLLEDTQIPSRLKFEV
EGRLKVNIDINQKAALQDLIYQDEIKYFYDKFPLSTIKVNISDTEIKPMTVDGLEGKIMTRVSNVEVLDIDNFIKKLSIL
LEYDYLNQNYQKNIQDFADNRLTTTKSQKDIIVQTCYAEVERVIKKHYEDVTGVFVKVIQNAFFFCLKSKIKLVPTSVIK
IEMDDKAVKYLAIITDFNPPEVINVLGALLELYSLHNTDIKSLMIEIDKSLNLNQRMIPNASGL
>lcl|NZ_CP026520.1_prot_WP_042234755.1_5346 [locus_tag=PC41400_RS26985] [protein=hypothetical protein] [protein_id=WP_042234755.1] [location=complement(5964613..5965101)] [gbkey=CDS]
MSDNKLNEDFKEEDNELLKESATLDEEEDIQDTIASLLYGNRLRDLVRSVAKHEGQNVMIGKKGINGMQNAMKFLTVMLS
YEILEVLKNKNKTVISPVITSEALNNMVGKSDSLNRVLLELEELKGRLHLLSGSTSISKATDYVNLFEREIINEHEVTDS
TD
>lcl|NZ_CP026520.1_prot_WP_042234757.1_5347 [locus_tag=PC41400_RS26990] [protein=ABC transporter permease] [protein_id=WP_042234757.1] [location=complement(5965327..5966187)] [gbkey=CDS]
MNTNAITAKKPITTKPPITAASVRKPLIRKPWFKKTLLTLVFWIAGLLLWHGLSLVYGPEILPGPILTAEGGYELLAEGT
LLQYIGISSYRVLLGWTLGSLIAIPVGLVIGKVGAIRAFAEPFLNFIRFIPPIAFITLFLVWFGIGETSKVVLILYATFF
IVVLNTLTGVLAVEEDKIRSARSMGASEWQIVLHVIIPATVPYMFTGIRLAMGTSFMAIIGAEMIASNEGVGYLIWNSRL
FFRTDWIFVGLICLGLMGFITDRAVALLGRKLLARYGVVGAVGGRR
>lcl|NZ_CP026520.1_prot_WP_042234759.1_5348 [locus_tag=PC41400_RS26995] [protein=ABC transporter ATP-binding protein] [protein_id=WP_042234759.1] [location=complement(5966203..5966985)] [gbkey=CDS]
MVTEAAPKRNEIHIERLDKSYTQPSAGELHYIIQDVNLAVRGGEFFVLLGPSGCGKSTLLSLIAGFVSKTSGQVRVGDRE
VDRPGQDRAVVFQQADSSLFPWLTVRENVEFGLRMKRVPAAERRSISDQYIELVGLKGHESKFPRELSGGMKQRVQLARV
LSNDPAILLMDEPFGALDAMTRRTMQKELQRIWQETDKTVIFVTHDIQEALLLGGRIGIMSAGPSSNISHIYDVPLAYPR
DLGSPEFYELYNRIQAHFDE
>lcl|NZ_CP026520.1_prot_WP_042234762.1_5349 [locus_tag=PC41400_RS27000] [protein=ABC transporter substrate-binding protein] [protein_id=WP_042234762.1] [location=complement(5967207..5968295)] [gbkey=CDS]
MRIMREAEEKGKKAAGHVGLRQTSRPHRYTRTLLALVLTAAMLAVLAGCGKAAADPKQGGAAKLKLKIADISTNSIFRVA
KSKGIFDKHGIDAELVSFATPAEGVNSLFIKQVDVAFGADFPLLNAVSKGDYSIFASSGTATDLSASEWKLFVRKDIKSA
ADLKGKKLSFSRGTFVPYLWDVYLGQNGVKLSEVKLIGQGGFDEAYVALKKGEIDGAWVYGAVLNEKFGALAEAHELTDM
SKTPVRLGNGLIAPTELLKANKAGFVNFLKAIDEASAYAQAHQEETADIMYKEVKQPKEATLKDLPKNPWKIGFTDKAYN
GYKEQKKYMVDNAIIEKDFDLKSKLYLDILREAFPDRVIEVK
>lcl|NZ_CP026520.1_prot_WP_042234763.1_5350 [locus_tag=PC41400_RS27005] [protein=aryl-sulfate sulfotransferase] [protein_id=WP_042234763.1] [location=complement(5968296..5969744)] [gbkey=CDS]
MGHPTIYPTGTTVYKPGKSWSGYTLYQAAETGALLIDMNGREVHLWKGLRGFPNKLLPGGYVFGSTAERDPKYGFQDEVN
LVQVDWDGNIVWKFDRYQHIEDPGHEPQWMARQHHDYQREGNPVGYYVPGQEPKVHSGNTIILAHKNVKRPEISDKELLD
DTILEVDWEGNVVWEWTTSDHFEELGFDEDAREALYNDPNTRFFGANAGDWMHINSASVLGPNKHYDAGDERFHPDNIIW
DARETNIIAITDKKTGKIVWKLGPNYNTEETKHIGWIIGQHHAHLIPRGLPGEGNILIFDNGGWGGYGAPNPESPDGLKN
ARRDHSRILEINPVTLEIEWQYTPTEAGFQAPLDSYRFYSPYISSAQRLPNGNTLITEGADGRIFEVTREHELVWEYISP
YKNKRNSNMVYRAYRVPYEWVPQLELPEEVEIAPIDVADFRLPGAAARGAVTTVEVEGTVSYGEGALCVARTDEAQPEPG
KG
>lcl|NZ_CP026520.1_prot_WP_042234766.1_5351 [locus_tag=PC41400_RS27010] [protein=MarR family winged helix-turn-helix transcriptional regulator] [protein_id=WP_042234766.1] [location=complement(5969816..5970526)] [gbkey=CDS]
MHEKDGALYEEADWLFRRLVRKFVKERDKISVEGIALPGMLILNSIHSQGEQRLGELAEQLDFTSGAVTALCDKLEAGGY
AVRKRSESDRRSVALDITDKGRELLRRNGGLSAHMIGILFGGFTPEELERQIVFSRRLLEKLDGFAEAVLAYPGEDGEAA
ANSSIKTTKRTADDRASTPVSFPASEEESPAAVFKNDTASHPSSELRPETSSQPLADQPPTPKSSKTTRKKNFISY
>lcl|NZ_CP026520.1_prot_WP_042234768.1_5352 [locus_tag=PC41400_RS27015] [protein=hypothetical protein] [protein_id=WP_042234768.1] [location=complement(5971144..5971662)] [gbkey=CDS]
MIDFLKIKNSKAYAEIMEIAERPGVLFSNDPLQMEAEAHYNMRTIERLNKIIKHDRLSLHELDNLFAFLEDAWASYLQKF
YAGKTFIFYLWGDYQIPAIRFSVISYEEGLGLPFACKVNQVSDRKEVLLAYREKACFDGIQILRHKEDSGTTNEEEEEAA
YTATVYSRIIEC
>lcl|NZ_CP026520.1_prot_WP_042234770.1_5353 [locus_tag=PC41400_RS27020] [protein=DUF4180 domain-containing protein] [protein_id=WP_042234770.1] [location=complement(5971659..5972024)] [gbkey=CDS]
MNITVDQRPHSKVAVISSEDVVISSVNDALDLMADIRYNHECDKVLLRKEQLTEDFFELKTKLAGDILQKYINYQLKIAI
VGDFSGYNSKSLNDFIYECNHGSNVFFKGTEAEALDALHGA
>lcl|NZ_CP026520.1_prot_WP_042234772.1_5354 [locus_tag=PC41400_RS27025] [protein=shikimate kinase] [protein_id=WP_042234772.1] [location=complement(5972067..5972618)] [gbkey=CDS]
MKRIHIIGSVGSGKTTLARRLSSELGLPFYELDNVVWKRTPAGDVRRTEEERDACLAGIIGADDWITEGAHQAGWIGPCL
EHAERIVFLDTPYFTRLRRIISRFIRQKTGQEQANYKPTWTILWKMFCWNHQFQREGRLKAMELLKTYGDKVLVLKDNRL
PHASILADAREGKQIGAQTDVLP
>lcl|NZ_CP026520.1_prot_WP_164977158.1_5355 [locus_tag=PC41400_RS27030] [protein=YxiJ family protein] [protein_id=WP_164977158.1] [location=complement(5972669..5973001)] [gbkey=CDS]
MDLSRPFPYKDLERIKQDFELELSLLSEEACLNADFNDYCVTVAGTISYVLSGSEENIPSRQMQLMKMNFFERFPSYKFF
ENKVSHYPAFQKELNSFEEARVLVLRYFIR
>lcl|NZ_CP026520.1_prot_WP_042234776.1_5356 [locus_tag=PC41400_RS27035] [protein=GNAT family protein] [protein_id=WP_042234776.1] [location=complement(5973120..5973656)] [gbkey=CDS]
MLFESSRVSFRKTTEEDAAVYHTWRNDMEVMRTTNPSLDLFTYAETRQFVEQVILGSGTSKSYMILDKLSERPIGITSLI
QLDFKNRSAECILDIGEKEFWGKGYGNESLRLLLNYAFHELNLHRVSLRVFAFNDKAVALYEKIGFKREGISRQALFREG
QWHDLLHMGILQEEYKAG
>lcl|NZ_CP026520.1_prot_WP_042234778.1_5357 [locus_tag=PC41400_RS27040] [protein=TetR family transcriptional regulator] [protein_id=WP_042234778.1] [location=complement(5973646..5974308)] [gbkey=CDS]
MSPKVNEEYKKQKKIDLLQKAKQVFIRKGYNGATMQDIMEAAGVSRGALYAYFDNIEHVYLELLRHEDRKHEIYFRSDDT
KNSWQQIKNWLNKQQAEIGSMEETLVLANSQFFSSAQYRNNKEIYPYIATRYEGTMEVLTGFFEKGAAQGEFALRLPAET
IARYLISFMDGLIMDTAHMGPEKTKVNEQLDALLFTLEAMLGPVGNKTETTEKENKEDVI
>lcl|NZ_CP026520.1_prot_WP_042234780.1_5358 [locus_tag=PC41400_RS27045] [protein=hypothetical protein] [protein_id=WP_042234780.1] [location=complement(5974596..5975036)] [gbkey=CDS]
MFKLKKIVPALVVTASLLLSSQSAFAVSGLADTRSEALNVYPNTDYTWTLDTAADEDWFYWTNSTSSTQYLSGSLLSPAG
KNYNLQLVLVTAGGWEYVLNTNDNGPGQHDGFNSTLQPGDKIYFKIRPSGFDYDKSNPYKFNFKVS
>lcl|NZ_CP026520.1_prot_WP_042234782.1_5359 [locus_tag=PC41400_RS27050] [protein=amino acid permease] [protein_id=WP_042234782.1] [location=complement(5975245..5976624)] [gbkey=CDS]
MEAKSLSRGLKPRHIELIALGGTIGVGLFMGSASTIKWAGPSVLLAYLLAGVIMFFVMRIMGEMLLLEPVTGSFATFAHR
YIGPLAGYVTAWSYWFLWVTVGMAEVTAIGIYVGYWFPDIPQWIPALIGVGIIAAANLAAVKFYGEFEFWFAMIKVVAIV
FMLVIGTAMIFFGLGNGGEPIGLSHLYSHGGFFAGGLKGFLFALCIVTAAYQGIEMVGITAGEAENPKHTLRKAIKNIIW
RILIFYVGAIFVIVTLYPWNQVGEIGSPFVMTFAKVGIVAAAGIINFVVLTAAMSGCNSGIYSAGRMLYTLAQNGQAPKF
FGKVSKSGVPRNSIITTISLLLIGVLFNYLAPDSKLFLYIYSASVLPGMIPWFALAFSQFKFRRKWKHEMADHPFKSRFF
PVSNYITIVFLFLVIIGMWFNPDTRVSLIVGAVFIAVVIVGYFVFGIGKRQKTGENEEG
>lcl|NZ_CP026520.1_prot_WP_042234784.1_5360 [locus_tag=PC41400_RS27055] [protein=iron-hydroxamate ABC transporter substrate-binding protein] [protein_id=WP_042234784.1] [location=complement(5976964..5977914)] [gbkey=CDS]
MKKFIPLSLFLILLLSACGGNQTGTKSGNSASTEPAGTASAATKSGTFTYQSETGPVEVPSNPQRVVVLTRTLTGNVMAL
GVPLVGVDEMSKENPNFEEKLKNVESVSDESLEKIIELKPDLIIGPSEIKNIDKFKKIAPTVTYTYGKVDYLKQQLEVGK
LLNKEKEAQAWVDDFTARTKKAGDEIKAKIGADATVSVIETFNKQLYVYGDNFGRGTEILYQAFGLRMPDKVKEATKKEG
YLALSTEVLKDYMGDYVIFSKNAEEDNSFQKTETYKSIPAVQNNRVFEANAKAFYFNDPLSMEYQLKFFTEHFLGK
>lcl|NZ_CP026520.1_prot_WP_042234786.1_5361 [locus_tag=PC41400_RS27060] [protein=TROVE domain-containing protein] [protein_id=WP_042234786.1] [location=complement(5978313..5979818)] [gbkey=CDS]
MSMAKKLFGALRPDAYNKDNYPAYTRSAEERYVQTLMTNTMGNTYYADQQELVGDAAQLHHEMAEANPHFMARAIVFARN
AGFMRLQPLFGLAILSMYRPDLFAKVFRQVVRIPSDLSDFLTILNGLGRGEGGRAVKRQVNRFLSEVSEYWAMKYNGRGR
GYSLGDAIATAHPKPKDLQQQALFRYLRGMEANLALLPQVEALENLKLAATEEEQIRWIERGKLPYETVTGAIKPSKAVW
EALLYQMPTFALLRHLNAMQRAGVLEDARHVEYIVQRLTDPQALSKAKILPFRLATAFRQVEHPELRDALREAAELTFDN
LPELGDQTAIFLDISGSMNGQYLEIGSVFALALYKKTRGSSLFWLFDTEVADAKPSRKDSILTQAAQIHARGGTDTGAPV
RKLIRERKKVDQIVIITDEQQNSGSPFYEELKRYRSKVNRDVKAFIIDIAPYRHAMVPVQDPHTYYIYGWSDTVLSYIAQ
TVQGYSGLTEQVATLDLDHLQ
>lcl|NZ_CP026520.1_prot_WP_042234788.1_5362 [locus_tag=PC41400_RS27065] [protein=hypothetical protein] [protein_id=WP_042234788.1] [location=complement(5979822..5980145)] [gbkey=CDS]
MVEQHVPVRPISSVVSTNENAERGARKGFFDLLGLSMTVSDGCREPFQPLSSLRFYPKKMNIRRGAGTGSFELIDQWQYP
GQYGRFDSCSRPVPLFLRRSFFEKRSV
>lcl|NZ_CP026520.1_prot_WP_042234790.1_5363 [locus_tag=PC41400_RS27070] [protein=hypothetical protein] [protein_id=WP_042234790.1] [location=complement(5980623..5981186)] [gbkey=CDS]
MSKRSKIGFLIGMAVVLLFVMSGCGSKEKTADVKPTETAKSSPASTANPANASAKPKEPAPESKLSEISNFMISSFWNEG
FVDITWYVSNGTGNTGKKLDIDFTMERLGKAMEKKAEYDMYIQGLDAKYDNVKQVWTKLSKETDNLYKQLKENTPKANDK
SYKFDTGLFEQYSDAFRDDVRALNKKQ
>lcl|NZ_CP026520.1_prot_WP_042234792.1_5364 [locus_tag=PC41400_RS27075] [protein=hypothetical protein] [protein_id=WP_042234792.1] [location=complement(5981305..5982018)] [gbkey=CDS]
MDKIETVLTKQGKTFKTYYHLMLTETTQVCIVNIVKGRLYADKKTANPKYEPYPTKGEAIKRMEELAGELKSKGFVEEPR
DVLFQIPEKEIVVFDKAKWHYEGDFPSELDVIQAYVPTGMFIAWLIKNDLTNKRSRKEDAPEIELVKRDEMTGAEFYRTN
WDGVLSSKELSEEAEAFAREYLDIQNDVYTAVDFTNVLAAGLPTIYHVEDSLENYRIIEPIISERYRDWKSRQGNLA
>lcl|NZ_CP026520.1_prot_WP_042234794.1_5365 [locus_tag=PC41400_RS27080] [protein=MBL fold metallo-hydrolase] [protein_id=WP_042234794.1] [location=complement(5982136..5982852)] [gbkey=CDS]
MELRRLTDRVFYTMFSKESDRPVLGYINGQTRSLMVDSGNSKFHAALFNEALRKEGLRKPEIVAITHWHWDHTFGMHAVE
GITIAHEKTNDKLAEQAKWEWTDIEMKKRLAAGVEIEFADTCIRNEYPSLSEIQVVTSDISFSGTMEIDLGGITVELHHV
TSPHSDDSVCIFVPQERVLFIGDAVGVDYYNDCHLDKDKLRSLLTAIEGFDFDMCVVGHAEPVSKLKTVAVMNSLLER
>lcl|NZ_CP026520.1_prot_WP_042234796.1_5366 [locus_tag=PC41400_RS27085] [protein=hypothetical protein] [protein_id=WP_042234796.1] [location=5983024..5983311] [gbkey=CDS]
MSRSVKKSPVFKDQQHLSTGWTKRQAGKAVRRFKGDVQNGKWYRKLYCPWNICDYRFYKTKQQAVHEWKTFQWLREQLLT
HAEVINDWEKFYRRK
>lcl|NZ_CP026520.1_prot_WP_042234799.1_5367 [locus_tag=PC41400_RS27090] [protein=hypothetical protein] [protein_id=WP_042234799.1] [location=complement(5983325..5984602)] [gbkey=CDS]
MHIQKFRYELTRGLGSIILYLKENPALTYRHLDAIADACIRNTAYDPQCDGSREAYLWEVIQLSKASPILQELILNALGG
TVNGWDLLQVYRLAKIFASHGNPTAVDAMKRGFRYDEEWNCFIGGEEIIEAAGANGFLFVAKEIGKRIMSSGYEGDTFVW
ESAKEFLGEEKVAALLEESVKDEKIQAFYQSVLGNEDDYSLLNGRKALSYEEFKSSIETGEKARYPYIVWGMRASKEDLF
KAADDLLKEENPKKLEAYLSIFVKPSFPLDPQKIIELAQSQNKRLRSAAIRALRNLKDERVHRLAVQLICQRETEVEALE
LFTFNYEENDLPLLERVAFKKHNKHSFHNMELDMVNIFEKHPTASCQKIMIELYHKGLCSPCRRRAVEIMLANDILPASL
RDEIRYDCNPDIRELWMKSASAGLP
>lcl|NZ_CP026520.1_prot_PC41400_RS30480_5368 [locus_tag=PC41400_RS30480] [protein=flagellin] [pseudo=true] [partial=5'] [location=<5984926..5985149] [gbkey=CDS]
RAMYGALQNRMEHTLNNVNNPAKNKMTGESRIRDADMAKEFRI**HPIC*RKRPKRYWPKQIRIHKPCYNYLSH
>lcl|NZ_CP026520.1_prot_PC41400_RS27100_5369 [locus_tag=PC41400_RS27100] [protein=DUF4256 domain-containing protein] [pseudo=true] [partial=5'] [location=complement(5985223..>5985321)] [gbkey=CDS]
GGTILCDRHDDTVFVQHNGAGSYYASRGFTAL
>lcl|NZ_CP026520.1_prot_WP_129112543.1_5370 [locus_tag=PC41400_RS27105] [protein=hypothetical protein] [protein_id=WP_129112543.1] [location=complement(5985619..5986728)] [gbkey=CDS]
MVIFMEGNPFKRFLARNVLLTHEVAEKLNVSRQRISILHKEGELVPIKSTANGSVYLLQDVLRFMEKKGMLPRYENPTPP
KYACETSGTHENIAYANKHIGQLNSIERVSIFFEQIDAAVENYFLPLEKYRYGDLISLSIPHMVIRDSDGNEMWLPGCNC
GYGGTGPHGSEKVLQLIGVPKALRENVFYHPVVKYVKNENGNWEVNVRKSEFDFRSISSIDDLEAAQANMYWHQGRLTLL
QDKRYSRSKPQHVLEKYWAFIPHPIEYILFSEDEQAIDNGFFDPSNQYGHKHGAYRLIIRDYSGRQLWLDPYIAKNKSLV
KQPELKDILEVCGFDIGKEKSDQKLSRWLELLLKKVEPSEPIVGRRHPF
>lcl|NZ_CP026520.1_prot_WP_042234804.1_5371 [locus_tag=PC41400_RS27110] [protein=AbiH family protein] [protein_id=WP_042234804.1] [location=complement(5987500..5990250)] [gbkey=CDS]
MIKEEGQEKVNRKNIVIVGNGFDISLGLKSSYQNFVEYIQSRKKITKDYELYNYNRLFLRKYANFHLNWSDFESLYEETV
RKINNRSLKNEEPQDIFDIVSVNDSIKRLEEDFHEYLSDEYPKWIEQKTILNAAGFKGFTEKVNPFFGKLVKDENTYFIN
FNYTNSIEDLCESVLYDANNISKSSLEVKKAKERVFHIHGSLEEGNILFGGGFTDSEDIKKIHYSQSLINDKIFRIKENE
SLNATRIKMMSDIDLDNNNTNIDLFIIGHSLQGSDFLFLSKFIQKANKIYIFYYEDDYILKMEEVLRKLGSSVAEKIVLV
PFLEILLQDKLFISSYEEYQTIDPFLKNKFPREDILSELSLTINHFTFRNINELRITTSNLEKVLQLSRNFVVNKNASRI
SRVYFDEKMDAVEFTKLSNSRDFLMVLKLVEKVIFSDTEIESVFFNQLLKYGNSLIYVKMEKCTFMVKEGNDSHANISAC
ESLRRFELIDCSCNYYKSNSSEIREGIFCFKSDTQNNIEKLKVLKNTNLVIENNVFEKSTNLIDLDISITEDVYQEEIHL
KNLESLQIENFVGFVPNITVGNKIKEIILIGYSVEFIKLSSIMRSNEESIGFPNFRLFHMKSPDQMTSCQNLIVDVILDF
FSNNTKFIVDEDIKSIKEYYLECRESYLEFFNPFEGLADKSILLKIKNLFEKSKVTDQSIIEEVLEEVSSKLEFKIQTIK
QMGQDVEQKDANGSNGSNQNNENDLERKFNNNDYIYTTIDGLLNDYATRNITLQELSDALQQESKAEILKEIYSGILNDI
KHLKFDSKKDVLNLKHKFFTQARDKVIRDFSEKWFVSKDELYLSAIQYYVGIDRIPNISGIIDSKDYESYKVKHSETNPL
KYAQAMKKDWKRELDEKVMWLNNELHNSEVLANNKI
>lcl|NZ_CP026520.1_prot_WP_042234806.1_5372 [locus_tag=PC41400_RS27115] [protein=HsdR family type I site-specific deoxyribonuclease] [protein_id=WP_042234806.1] [location=complement(5990303..5993464)] [gbkey=CDS]
MTKVKHNTEAEVERRLIEVLGQGHNQWQYRPDLKSEDDLWKNLRQKITQNNLSEIGDHPISDKEFDTIKTELLLKTQTPF
DAAKWLKGENGIARITIDREDVFLGSMSLVLYSNQSIGGGISTYEVVHQIAKQKANLNDRDRRFDVTLLINGLPIVQIEL
KQVSAKDGVFQAFNQIKKYAEEGMFRNNIFSTLQLFVISNEQTTRYFANAMPKDMHKKLIFSWRTTENRKVENLYEFVKQ
VLNIPDVHRLIANYTIVSEDQDNKALIVLKPYQVHAIQALFKSAMKHESGYVWHATGSGKTLTSFVATKLLARKTGVDRT
IMLIDRKDLDNQTTAEFTKFASEFNTGISSGNAKSNSLIVGTGSAKELSQTLLSDANSNTVIITTRQKLEAALRYTKKQE
DPKAKQRFDKLLGQHIVFVVDECHRALSSEGMKEIKNFFPNSTWFGFTGTPIFEENKKQAKGELARTTRDQYGERLHTYT
IKNALDDGAVLGFQVEHENTIESISLNNTIFKQLRASEKYTALSNDEINAMIDEMDEIEKEDYLELSTFEQDDHIQKVIH
KIFKPDNAYIKFDFQNGRPQKSAILTTSSIGMAKRYYNAIKEMTNDPEWLKNEFAGHPIRAGRTIDDPDFPRIAITYSIQ
ENEENSKQVHEEMKDVIKEYNEYYHTAWSIEDIERYNGDINNRLARKKAEFKQYGNHIDLVIVVDRLLTGFDAPTIQTLF
VDRNLSYANLIQAFSRTNRTYSGKKKGLIVTFRKPATMERNVQVATKLYSHEQEETNLVYPTYKETKKRFTKAHNKLTTL
VPDPIEINEHSPLETRIEFVKAFQELNNAYEALATYDDYNDEMGKSKNLQEQVKALEEYIGVYNTIKGSLIEPKDPDNEP
EPDFSDIEFYGENAVKIYDIDSTYIDRLLETYSANNQNTREEIEKALQKLKKSENVKDVYRAMLNAIDTGEVDSWEDILV
VKRRFFTETVNEAVQKFADTWFVDESELHSSAIQYVIGADPIPNIGNIINSKDFDKYKAANPGAKPLKYGPEMKRQWRKT
LDEVIVPLNDELR
>lcl|NZ_CP026520.1_prot_WP_042234816.1_5373 [locus_tag=PC41400_RS30485] [protein=restriction endonuclease subunit S] [protein_id=WP_042234816.1] [location=complement(5993488..5994711)] [gbkey=CDS]
MGNKSKPEIQFPGFIEEWEKRELRGFVQDYIERTVVQNQYPVLTSSQKRGIIYQEEYFADRQITTQENIGYFVLPRGYFT
FRSRSDNGVFKFNRNDLVDKGIISYFYPVFKIINGNSDFFLNMLNSTIKKQVALEAEGTGQKVLSLNKFKNINVVIPEIQ
EQTQIGNFFKQLDDTITFYEQELTTLQQTKQGFLQKMFPKEGESVPEVRFPGFAGDWEQRSLKEVTKKIGSGKTPKGGSS
NYKLQGIPLLRSQNIFNDKVNLLDVVYISEEVDEDMANSRVQQNDVLLNITGASIGRSAVYEFETSANVNQHVSIIRPYE
NVNPYFIQLNLTSGNGQNQIELNQAGGGREGLNFQQISKMKFYFPNLKEQVRIAGFFKKLDQVISIHQRELELLKQTQKA
FLQKMFV
>lcl|NZ_CP026520.1_prot_WP_042234819.1_5374 [locus_tag=PC41400_RS27130] [protein=type I restriction-modification system subunit M] [protein_id=WP_042234819.1] [location=complement(5994701..5996293)] [gbkey=CDS]
MADLNTRLFSAADNLRSKMDASEYKNYLLGLIFYKYLSDKLLEKVVEIADESLEEYSTQENQTQLYRNLLADEEIKSDLI
ETLVDTLGYDIEPDYLFSVLANQAKQNTFQLNDLSAAFTNLSSKYDQFNGLFDDVDLRSKKLGSDDQQRNITITEVLKKL
NDIDVIRHDGDVIGDAYEFLISQFASEAGKKAGEFYTPHQVSVMMARIVAMGQEDKKLFSVYDPTMGSGSLMLNVRNYIN
HPDSVKYHGQELNTTTFNLAKMNLILHGVDKEDMRLRNGDTLNKDWPTDEPYTFDSVLMNPPYSAKWSSDDTFLDDSRFN
RYGKLAPKSKADFAFLLHGFYHLKETGTMAIVLPHGVLFRGAAEGVIRKKLLEDGSIYAVIGMPANLFFGTSIPTTVIIL
KKNRNMRDVLFIDASKEFVKGKNQNKLSDENIDKIVETYMKRENVGQYAYLASFDDIKENDFNLNIPRYVDTFEEEESVD
MAIIGSTIKNIRKQKAELESSLYDIISSLQYDEENAEWIQGALEVFNRGK
>lcl|NZ_CP026520.1_prot_WP_241688746.1_5375 [locus_tag=PC41400_RS30490] [protein=DUF5655 domain-containing protein] [protein_id=WP_241688746.1] [location=complement(5996355..5996903)] [gbkey=CDS]
MPRLICVAGDFTKYDEYAVKQINRNIDLIRYKKFGADLLMLELVNSSTVKSFNIQEQGDKWIKPSSDKTFQERLQSTTPM
LRELYYSFRDYILAYGDDVTENQLKLYTAFKKNKTIICVEVYQKQILWHIRLYPDDIELEEGFTRDMRNVGHFGTGDLQI
IIKSPEDIEKAKPLIDKAYNSN
>lcl|NZ_CP026520.1_prot_WP_241688747.1_5376 [gene=rlmD] [locus_tag=PC41400_RS27140] [protein=23S rRNA (uracil(1939)-C(5))-methyltransferase RlmD] [protein_id=WP_241688747.1] [location=complement(5997491..5999881)] [gbkey=CDS]
MSKNKRRGGASAGKPQGGALSGQKRGGMSAGRRQGEASRGQNLGESERQGAASLGQNRDGTSMGKTQGGPGDRRGTLRDE
NATKAGFTADGRSEVNRTHTGQGRGGAAQRAFGGRGEAGRVKPYGEAEAARGERAGSRESGGRARAERTSAPQAGSKRSA
ALAALPVEKNGEYTAEIIGIGHDGEGVGRVEGFTLFVPGALPGERVRLKVLKLKKQYGYAKLLEVLESSPDRVGAPCPIY
DKCGGCQLQHLDYAAQLKVKRQLVVDNLERIGKLRVAKEADAEAGNAVGDASVLRADTVGGNTTRPSEGFKPADEEITVP
EDKVINDEALESKEDADDTISGSGLQPLMDTEFVDASDLLEDESDGNLAGDSSAQAQTNDSENITWDNTTGDNTIGGNAA
LGIGAATGTSPIYDVSEDIGESTLAAGVKSTNLRRGESTEPLKQERVCASEADGIVVYPTIGMSDPWRYRNKAQVPFGEE
QGGLVGGFYAQGSHRIIDMETCLIQHANNDEVIAKVKEIGRGLGIRAYQEETHEGLLRHVVVKVGFRTGEIMVVLVTNGK
EVPHVAEWIEQIRAAIPGVASICQNVNTGRTNVIFGATTRVLWGREVIYDYIGDVKFAISARSFFQVNPVQTEALYGKAL
EYAGLTGSETVVDAYCGIGTISLFLAKHAKKVYGVEIVPEAIEDAKRNAELNGIRNAEFAVGGAEDVLPEWQRAGIRPDV
IVVDPPRKGCDERLLDTILQLRPERVVYVSCNASTLARDLRVLENGGYRTVCVQPVDMFPHTVHVESVALMSRVEK
>lcl|NZ_CP026520.1_prot_WP_042234821.1_5377 [locus_tag=PC41400_RS27145] [protein=diacylglycerol kinase] [protein_id=WP_042234821.1] [location=complement(5999887..6000816)] [gbkey=CDS]
MAKRARLIYNPTSGREEMRKRLPEVLQRLERGGYETSTHATIGEGDAMLAASQAVDRGFDLIIAAGGDGTLYEVVNGMGE
KPNRPPVGILPLGTTNDFARALNIPRDWEDAVDLIVRGYTRPIDVGKVNQKYFINIAGGGSMTELTYEVPSKLKTMIGQL
AYYVKGLEMLPRLRPIHMHLKSDEMEMNEEVMLFLITNSNSVGGFEKLAVDASLNDGLFDVIVLKKCNLPEFIKIATMAL
KGEHTNDPNVVYFQTRKLEISSPDYVQINLDGELGGTLPAVFTNLQSHIEIIVDESGLSTYKHQRRLGS
>lcl|NZ_CP026520.1_prot_WP_042234858.1_5378 [locus_tag=PC41400_RS27150] [protein=CbiX/SirB N-terminal domain-containing protein] [protein_id=WP_042234858.1] [location=complement(6000826..6001614)] [gbkey=CDS]
MRTYGILVISHGSRSREWVQLVDEAVEGMRVPDGVPVFGSFLELVDGRLIQDGITWLENRGVTDLIVIPLFVSKGSTHLD
EIAYALGLKPEPLLPTDLEPFEIKASVHMTPPMDEDAELLAEMVYEKIRNLSKDPEREIVLLVGHGSVEQLFHLAWRGSL
ERIAQALKKTGGFGTTDVAMLLPDQTKRKLEWWQERKPGHAVIIAPLFLSEGYFTREVIPSRLEGFDYRYNGQSILPSPK
VSEWLERQVASFFNRDHYEQKR
>lcl|NZ_CP026520.1_prot_WP_009675256.1_5379 [locus_tag=PC41400_RS27155] [protein=YerC/YecD family TrpR-related protein] [protein_id=WP_009675256.1] [location=complement(6002398..6002688)] [gbkey=CDS]
MQLKKLNDKAIDQLFEAILTLNSLEECYTFFDDLCTVNEIQSLSQRLEVARMLRKGCTYNQIEAETGASTATISRVKRCL
NYGNDGYVLTLDRLNR
>lcl|NZ_CP026520.1_prot_WP_042234824.1_5380 [locus_tag=PC41400_RS27160] [protein=hypothetical protein] [protein_id=WP_042234824.1] [location=6002975..6003601] [gbkey=CDS]
MSYRYPKHGRQPEVQTRSLFSPGPAGGQGGVPFQGGGVPGGGFSDGGFPGGGFSGGGFPGGGFSGGGFPGGGGFPQGGPG
FSGFGQNLPVPVNSAAPSAAPAAAAGSSGGGGAGGLLGGMNINQIKGFIDRMGGIEGVIGTMGKVQKIVGSVQQFAPMLK
LMFSGFGAKAATKRANTKPSAKRKKKRKSTQTKRKVKTTKSRKQSSRK
>lcl|NZ_CP026520.1_prot_WP_042234827.1_5381 [locus_tag=PC41400_RS27165] [protein=hypothetical protein] [protein_id=WP_042234827.1] [location=complement(6003906..6004088)] [gbkey=CDS]
MKAAVAEGLGPLKRLKEALWLASQSTAPYHAKVVSGARCFLDGKSKADRMLFKYFSVYFM
>lcl|NZ_CP026520.1_prot_WP_042234829.1_5382 [locus_tag=PC41400_RS27170] [protein=thiol-disulfide oxidoreductase DCC family protein] [protein_id=WP_042234829.1] [location=6004156..6004584] [gbkey=CDS]
MKAPTGGPEAISGPVVLFDGVCNLCNAAVRFTVRRDPAGQVRFAALQSEAGQRLLARHGLPAASFDSFVLIEGGRVYTQS
SAGLRVLRRLRYPWPLLYGFVAVPKPIRDAVYRWIARNRYRWFGQRDVCMVPTPELRRRFLD
>lcl|NZ_CP026520.1_prot_WP_042234832.1_5383 [locus_tag=PC41400_RS27175] [protein=membrane-binding protein] [protein_id=WP_042234832.1] [location=complement(6004656..6005882)] [gbkey=CDS]
MRKLVYGLTMMLALFLLPLAASAASGTQLQVNDKAIGLSAEPQISGDTAYVDLAPVLGSLGYTVQQSVYGTGWTVSKDGS
KIELVSGSSSFVVNGESVKAGAAVIQGGNKLNLQAFADLTGRKLKVDGEKGLISLNDTLQTSVLETLYTGKLTSSAQAPK
ALTQGNQGYTGYIKIFYPNGKVFYDGYMVNSKFEGKGKYYDSNGVLRYDGNWKDNKFHGQGKSYYADKHLEYDGMWKAGL
KQGYGTYYWAWEPCGEPPGSITGYNKYVGFFDKDEFDGVGKIDWADGSKFEGFFSKGQIKCAGVFYWTDGWKYVGMYKDG
KRDGFGTEYNEKGTVIYEGGFRQNCAEGYGRLYDANGNLLFEGEFVNGHPKSEATARTLTGFQAQSNQKSSEAQLQTHEQ
NQMEATFQ
>lcl|NZ_CP026520.1_prot_WP_164977159.1_5384 [locus_tag=PC41400_RS30015] [protein=hypothetical protein] [protein_id=WP_164977159.1] [location=6006192..6006344] [gbkey=CDS]
MGTIWTFICGHWISIVMILGCLIAIYVVYSKRNELQLYWLNKREARKRRS
>lcl|NZ_CP026520.1_prot_WP_042234834.1_5385 [gene=purD] [locus_tag=PC41400_RS27180] [protein=phosphoribosylamine--glycine ligase] [protein_id=WP_042234834.1] [location=complement(6006546..6007814)] [gbkey=CDS]
MRILVIGRGGREHAIVWSLRKSPKVTALFCAPGNGGIAELAECVPITEFQFAEIAEFVRQNAIDLVVVGPDDPLYEGIVD
YLEEHGIRAYGPNRAAAEIEGSKVFMKHLLKKYEIPTAAYEAFEDYEAALAYLRKLGAPIVIKADGLAAGKGVIVASTLE
EAEKALKEIMVEKQFGASGNRVVIEEFLQGQEMSLLAFVDGNVVKPMVPSQDHKPVFDGDKGPNTGGMGTYSPVPHIPQE
IVQQALDTIVKPAAEAMVKEGRPFRGVLYAGLMITADGPKTIEFNARFGDPETQVILPRLKTDLLEIILATLEGRLADLA
IEWSEEAAVCVILASGGYPGSYPKGLPIEGLDAVGGALVFHAGTALEDGRLVTSGGRVLGVTALGRDIADAREKAYAAAD
RISFEGKQNRTDIALKALNAMK
>lcl|NZ_CP026520.1_prot_WP_042234836.1_5386 [gene=purH] [locus_tag=PC41400_RS27185] [protein=bifunctional phosphoribosylaminoimidazolecarboxamide formyltransferase/IMP cyclohydrolase] [protein_id=WP_042234836.1] [location=complement(6007834..6009378)] [gbkey=CDS]
MAIKRALISVSDKTGIVEFASELAKLGVELISTGGTKTLLEKEGVPVIGISDVTGFPEILDGRVKTLHPAVHSGLLAIRD
DAEHRKVMEELGLGYIDLVIVNLYPFAQTIAKPDVEYADAVENIDIGGPTMLRSAAKNHAFVSVVVDAADYDTVLTEVRE
NADTTLETRKRLAAKVFRHTAAYDSLISEYLSGQVGEEFPERYTVTYEKVQDLRYGENPHQKAAFYRKPLAGEGSIATAE
QIHGKELSYNNINDANAALQIVKEFTEPAVVAVKHMNPCGVGIGTSVKEAYEKAYAADPTSIFGGIIAANRPIDGETAQL
LHEIFLEIVIAPDFTAEALEILQKKKNIRLLKLGNFTEGQRKALHLVTSVEGGMLVQESDVHAISEGDLKTATDREPTEA
ELKQLLLAWKVVKHVKSNAIVLVTEDMTIGIGAGQMNRVGAAKIAIEQAGEKAKGAVLASDAFFPMGDTLEIAAAAGITA
VIQPGGSIKDEESIKVANDKGIAMVMTGVRHFKH
>lcl|NZ_CP026520.1_prot_WP_042234838.1_5387 [gene=purN] [locus_tag=PC41400_RS27190] [protein=phosphoribosylglycinamide formyltransferase] [protein_id=WP_042234838.1] [location=complement(6009469..6010104)] [gbkey=CDS]
MGQLRIAVFASGSGSNFQAVADAVRGGRLEAELALLVSDRPQSKVVERAKQAEVPVFAFSPKAYDSREAYETEILQLLRE
KEIDLIVLAGYMRILTPVLVEPFYGRMINVHPSLLPAFPGIKGIAQALEYGVKVTGVTVHYVDGGLDSGPIIAQRALEIR
EDDTVESLTERVHAAEHELLPLAIGWIAEDRVTLEGRLVRISEPQTAKAAE
>lcl|NZ_CP026520.1_prot_WP_042234840.1_5388 [gene=purM] [locus_tag=PC41400_RS27195] [protein=phosphoribosylformylglycinamidine cyclo-ligase] [protein_id=WP_042234840.1] [location=complement(6010106..6011146)] [gbkey=CDS]
MSEAYKKAGVDIAAGNEAVERMKKHVQRTFRPEVLTGLGGFGSLFSLNKDKYEEPVLVSGTDGVGTKLKIAFAMDKHDTI
GIDAVAMCVNDIVVQGAEPLFFLDYLACDRVIPEKIEAIVKGIADGCAESGCSLIGGETAEMPGMYTEGEYDIAGFTVGI
VDKKKIIDGSTVRPGDAVIGLASSGVHSNGFSLVRKLLLEEKGHNLTDTIDELGGVLGDVLLEPTKLYVKPVLGILEQVG
IKGAAHITGGGFLENIPRTLPEGVNVNIQYGSWPILPIFSLMQREGNITNNDMFRTFNMGIGMVIIVAQEDAEKALEIAA
SYGHDAYRIGEVTEGGKIVTFEGAEV
>lcl|NZ_CP026520.1_prot_WP_042234861.1_5389 [gene=purF] [locus_tag=PC41400_RS27200] [protein=amidophosphoribosyltransferase] [protein_id=WP_042234861.1] [location=complement(6011270..6012751)] [gbkey=CDS]
MSDAVTSGKGLWTGSYYNEGHGGHDGLFDKLKEECGVFGVFGHPDAASLSYYGLHALQHRGQESAGICTADADNFHYFRG
MGLVKEVFDQSNLPTLVGTSAIAHVRYSTAGESKLANAQPLVFKYREGDLAVATNGNLVNAGEIRQQLEKQGSIFQTTSD
TEVVAHLIARSKHDDIVDAVKDALQQVIGAYAFLILTKDKLIAALDPNGLRPFAMGKLGDAYLFASESCAFDIVGGTYMR
DLLPGELLVLDRNGLTEDRFTESQTRATCAMEYIYFARPDSDIDAINVHSARKRMGKRLASEAFVDADLVTGVPDSSTSA
AIGFAEQTGIPYELGLIKNRYTGRTFIQPSQELREQGVKMKLSAVRKVVEGKRVVMIDDSIVRGTTSLRIVNLLREAGAT
EVHVRISSPPFQNPCFYGIDTPSRKDLIAAQKTIEEIRQAINADSLYFLTQEGLIDSIGRKDGVENGGFCTGCFDNRYVT
YVNDSENAVGCGC
>lcl|NZ_CP026520.1_prot_WP_042234843.1_5390 [gene=purL] [locus_tag=PC41400_RS27205] [protein=phosphoribosylformylglycinamidine synthase subunit PurL] [protein_id=WP_042234843.1] [location=complement(6012736..6014979)] [gbkey=CDS]
MAQQLAAKEPNAEQIAEQKIYKQMGVTDAEYDQICGFLGRKPNYVEIGVFSVMWSEHCSYKNSKPLLRRFPTTGERVLMG
PGEGAGIVDIGDNQAVVFKIESHNHPSAIEPYQGAATGVGGIIRDIFSMGARPVALLNSLRFGRLQNDRVKYLFEHVVSG
IAGYGNCIGIPTVGGEVTFDESYEGNPLVNAMCVGLIDHDKIQRGVAKGVGNPVYYVGPATGRDGIHGATFASEELTSES
EAKRPAVQVGDPFMEKLVMESCLELIDTGIVLGIQDMGAAGLTCSSAEMASKAGNGLELYLDEVPQREEGMTPYEMMLSE
SQERMLFVIDPKDEAQALEIFERWGVICAKVGKVTDDGRLKLFHQGQIVGDMPVTALVDECPVYNKPSAVPAYYNEQANT
DTTRYNEVTDLNGALKSILGSPTVASKEWVYNQYDYMVRTSTAVQPGSDAAVVTIRGTRKALAMTTDCNSRYVYLDPEVG
GRIAVSEAARNIVCSGGEPLAVTDNLNFGSPEKPEIFWQLEKAVDGMAEACRELSTPVIGGNVSLYNENAKGAIYPTPVV
GMVGLIQDTDHITTQGFKNEGDVIVLLGETKAELGGSEFQKVVHGVVEGRPPQLDLETEKRLHAGVLTAIREGLVASAHD
LSEGGLAAAVAESSISGGLGADVDFRTELRADHALFSESQSRILLSCRPEKREALIALLQEKNVQAEVIGTVTGTNVKVD
INGKPAVQASISELRQVWKDAIPCLMQ
>lcl|NZ_CP026520.1_prot_WP_042234845.1_5391 [gene=purQ] [locus_tag=PC41400_RS27210] [protein=phosphoribosylformylglycinamidine synthase subunit PurQ] [protein_id=WP_042234845.1] [location=complement(6014957..6015652)] [gbkey=CDS]
MKFAVLVYPGSNCDIDCYKAVEDTVGEPVDYVWHTATDLSEYDCILVPGGFSYGDYLRCGAIARFASVTQALVKAAEQGK
YILGICNGFQVLTEAGLLPGALLRNRSLKFRCHSTLLKVVNNATPFTSDYAEGEVINIPIAHGEGNYYCDEETLRELQQN
NQIVFRYETDTNPNGSVDDIAGICSKEGNIVGMMPHPERAVDELLGSVDGKRMFTSILNAWREQHGTAVSR
>lcl|NZ_CP026520.1_prot_WP_009675257.1_5392 [gene=purS] [locus_tag=PC41400_RS27215] [protein=phosphoribosylformylglycinamidine synthase subunit PurS] [protein_id=WP_009675257.1] [location=complement(6015673..6015912)] [gbkey=CDS]
MKATVYVTIKQSVLDPQGNAVQGALHSMGFEEVGKVRVGKYLELELDTTDRAEAEERLKAMCEKLLANTVIEDYRFELN
>lcl|NZ_CP026520.1_prot_WP_009675271.1_5393 [locus_tag=PC41400_RS27220] [protein=phosphoribosylaminoimidazolesuccinocarboxamide synthase] [protein_id=WP_009675271.1] [location=complement(6016259..6017152)] [gbkey=CDS]
MGASVALSTAEHLIKAPLVYKGKVRELYDLGEHFLIVVTDRISAFDYVLDPAVPDKGNVLNTLSKFWFELTTPFMENHVV
HCDVELLGDLVTEPALLKDRVMVTKKATRIPVECVVRGYITGGGWRQYQETGMINGIRLPEGLRKNQVLEHPIFTPAAKN
DVGHDEDITYDQMCELVGEKVTEQLMTKSVMLYTLAHTYCEHRGIILADTKFEFGFIGNELILIDEIFTPDSSRYWAKDK
YELDTEIDSMDKEPVRTYLANSGWDKNSAPDPLPDSVVEETSRRYREILQRLTEETE
>lcl|NZ_CP026520.1_prot_WP_042234849.1_5394 [gene=purB] [locus_tag=PC41400_RS27225] [protein=adenylosuccinate lyase] [protein_id=WP_042234849.1] [location=complement(6017183..6018478)] [gbkey=CDS]
MIERYSRPEMVNIWTEENKFKAWLEVEILSCEAWSELGVIPKEDVKELRAKADFDMDRIYEIEQETRHDVIAFTRAVSEK
LGPERKWVHYGLTSTDVVDTALGYLLKQANEILEKDLLNFIGILKNKAIEHKHTAMMGRTHGVHAEPTTFGLKMALWYEE
MKRNLERFRFAADGVQYGKISGAVGTYANIDPFVERYVCERLGTNPAPISTQTLQRDRHAEYMATLALIATSLDKFATEV
RALQKSEFREVEEAFAKGQKGSSAMPHKRNPIGSENISGLSRVIRGHMVSAYENVTLWHERDISHSSVERIILPDATILL
NYMLNRFGNIVKNLTVFPENMKRNMQSTYGVPFSGRVMTKLIDKGFSREQAYDTVQPRAMQAWEEQRSFREIVENTAAIT
ELLSAEEIEDCFNPSWHLKHVDTIFDRLGLK
>lcl|NZ_CP026520.1_prot_WP_042234863.1_5395 [gene=purK] [locus_tag=PC41400_RS27230] [protein=5-(carboxyamino)imidazole ribonucleotide synthase] [protein_id=WP_042234863.1] [location=complement(6018641..6019840)] [gbkey=CDS]
MSMEKAAAHATPEQHKVIPPGSTIGVLGGGQLGRMLALAGRNMGYRFVTLEPGADSPCGQVADRQIQASYGDVDAARELA
RVSDVITYEFENVDAAVTKLLTQEAHVPQGSALLYTTQHRLREKRAIEAAGVRVAPYAEITSEEDLRAAVGKFGTPCVLK
TATGGYDGKGQWVIRSADEIPDAFAELSRAGTDLVLEQFIRFEKEISVIAARSPRGEVKAFPAAENIHVENILHMSIVPA
RISASVQEEAERLAIRIAESMKAVGLIAVEMFLTKDGELFVNELAPRPHNSGHYTMEACLTSQFEQHVRAVCNLPLGPTE
LLTPVVMVNVLGEHMEPLLKWFCSEESEASDSGVTPKLHLYGKDEAKTKRKMGHINLLTSDTAKAAEWMERTGIWDSIS
>lcl|NZ_CP026520.1_prot_WP_042234851.1_5396 [gene=purE] [locus_tag=PC41400_RS27235] [protein=5-(carboxyamino)imidazole ribonucleotide mutase] [protein_id=WP_042234851.1] [location=complement(6019954..6020439)] [gbkey=CDS]
MSARVGVIMGSKSDWETMKHACDVLDEFGIPYEKKVVSAHRTPDLMFQYAEQAAERGLRIIIAGAGGAAHLPGMVAAKTE
LPVIGVPVKSRALNGLDSLLSIVQMPAGIPVATVAIGEAGAANAGLLAAQILGAFEPELQNKVRERRERIKAQVLEASEL
E
>lcl|NZ_CP026520.1_prot_WP_042234983.1_5397 [locus_tag=PC41400_RS27255] [protein=DUF5704 domain-containing protein] [protein_id=WP_042234983.1] [location=complement(6026522..6030085)] [gbkey=CDS]
MKRRHVLPLILLCVQLFFVLVFLPTSIKANEPPKSQRNSTSKEIISDVVMAKPGVYWYQIDNGKFRADYSSGQKQKDVEW
NKVCDNPFPVKSSMEDDVNLSSWKPQDWEYNGRTIPRDKVDKVRLYEVTYHDAFSYEPVIPGSKPDIINDSYALLNTYTG
MDYIESDYKEFMKRPNGCPKVVVMYETPVDITWAGDIYEEKEITVIPDSSISVGQKIQLQANVKTKDWGASDWGKDYDVA
TRTTETTWKSEDETIATVNASGQVLGKKAGKVKVRAIWDNGDYRISDAAEITVTTDPGLVINLPKPEFCTSDSSIQQAEA
VLTKPDGTSWPLQKHDKLTWTSSNPSIATIDQTGKITLKAVIGTTEITAHFKDDLQHLDEKKSVTLTIKDCGSSGGGNPG
TGNPQPPGGTNGCNPVINPPSKGLMQSGYFMDPQASGMLRADRRGAETFNVLEGIPTSESLYANVQSLHYLFQNKFTNMT
GEVTYSVPVTKTYVWTVPVPPPGVPIPMSQTVTQTMTVKRPYGYWQIDNLEIYRPEQARLSNYALGGYGGSVTLDAKNYT
PPVITSTNKDDVSSHVQPSNCNSVNLGTGGGPPMNETGVFQAAAEAAVGANKVSNDLLVFNGNTLMDNRIHDAAAPVPKP
IPEPARIGPDIFYGTGYIISKTLANRQAQPTTGTIAYTLLPGNIKGGVNKTFDIPGINPVTVHTPVIMVPSVSDDQAHNQ
KTTPAYDRSALILDRPFSVTVPTTGPHRGIPGYGIRDYAKYTRQKQVWFPFDTYDAAMKFIPKRTWIDIPVGQLSTTFYM
PVWVDEGPYSVLFRSISENAPASFTTEPQANLTLSNHVATDTVPVDVIGRLYDFKVTDIADFNWENVFRTAKGSAAHTGT
KYWIGEKDIDGHPRPTGYPFILPIHPGSHPEAGYKNISVKTGYHIKFDLKTKGNMFGFDDGVRITPAFYFVSNEGGKRQP
VDLYYHADNKPFVRIGSPQDQEKRFVVLNDRLRNLSALELEQTAGYLFDQDPGSFTNRAEFTADYIKKAAKPTWVGTYHW
LILSRQVRTFIGETQNIPAGVDRARALASDQKWYGEYSLPAAVYAVPKGTDLAVYGRKQTLDDQSPVFLRNGYIVVNFNI
ESIRGGDVNKPYLQYIHGPLNNQWQKEGGVGSVTGTKGQSFPVMDGDVVFYHGDLSSYDDFSSNGTH
>lcl|NZ_CP026520.1_prot_WP_042234981.1_5398 [locus_tag=PC41400_RS27260] [protein=S-layer homology domain-containing protein] [protein_id=WP_042234981.1] [location=complement(6030105..6031382)] [gbkey=CDS]
MVFLRKNKRFWGAAILFGTAAILLHNDAKAVGLTTELAAAVPVQSFSDVGGHWAETVIRKAADLNLVQGYSDGTFHPDGK
ITRAEFAAMLNRATKQKTDTAGAENPLSGLQNHWSQAEVNKLAALGFIDQSDYKTGFSPDLELTRYEMIKWISSGLAQSE
PSFKQALEDTEGTLLPTPETYKGGISSEQIPSVALVKGTGIVDGFEDGTFRLAQATTRAEVSAILMRFMEVEGNKAEQYK
GLNELREVGLTGSNLTTITKYKYGMDYNGVVRDFSFIRGISISLRDGIGTMKINRVIAVDSDNDSNVSIYKPMFLENVEK
YQKNKYVVFSEIAVQSSVKNLTNLELGNSMGNGITSSTRLLGDNVIRFGLKTLSNRNNNELLNSGNDEFFWVSEVIPKTV
PENYPKLSFSIKAQNDDLVRIILNK
>lcl|NZ_CP026520.1_prot_WP_042234979.1_5399 [locus_tag=PC41400_RS27265] [protein=NCS2 family permease] [protein_id=WP_042234979.1] [location=complement(6031672..6033078)] [gbkey=CDS]
MDRFFRLKENGSNVRTEIMAGITTFMAMAYVLVINPNILTAFGKTGMEWYPVFLATALAAGIFTIAMGLFVNFPVALAPG
MGLNAYFATVVVSTASTNNPITWQMALTAVFISGIIFIILTITRLRQMLLVAIPDGIKHAITVGIGLFITIVGLKNTGLM
TIAAETVNDIPKGQFTPLKGFETVIELGSFKSPDVIMALIGLVLISILMVLRVRGAILFGILGTTIIALLMGYVDLSSLN
NPQTPWIPDLGQLNFWEFDFAGIMGVGIVSVIVTFTFVEMFDTFGTLVGTANRAGIMKNKEEGNKRVGKAMFVDAFAVSG
GAMLGTSTVTAYVESSAGVAEGGRTGLTSLTTGILFILALFFAPIIALIPGSATGAALIVVGVLMMQAVRDIDFQDLVIG
IPAFLTITFMPFTYSIANGISLGIVGYVILAVISRLSSRVEKKHSIHWLMWVLFVLVIARYVFIGVSG
>lcl|NZ_CP026520.1_prot_WP_042234995.1_5400 [gene=guaA] [locus_tag=PC41400_RS27270] [protein=glutamine-hydrolyzing GMP synthase] [protein_id=WP_042234995.1] [location=complement(6033535..6035073)] [gbkey=CDS]
MQKTHELIVVLDFGGQYNQLIARRIRDLGVYSELLPYNTSADKIRELNPKGIVFSGGPVSVYGEGSPVVDPAVYDLGIPI
MGICYGMQMIAHQLNGKVERAATREYGKSEVQFQSHSRLAKNLDSKQVVWMSHGDHVCDLPEGFVLDASTETCPIAAMSH
PDKNIVAVQFHPEVRHSAQGNEMIRNFIFDICDCTGDWSMESFIEDTIRDIREQVGDRKVLCALSGGVDSSVVAILIHKA
IGDQLTCMFIDHGLLRKDEAESVMETFVGKFDMHVVKIDARDRFLGKLAGVSDPEQKRKIIGNEFIRVFEEESAQLGDFA
FLAQGTLYTDIVESGTATAHTIKSHHNVGGLPEDMKFQLVEPLKALFKDEVRKVGEECGLPGAIVWRQPFPGPGLAIRVL
GEVTEEKLKIVRDSDAILRDEIAKAGLDREIWQYFTALPNMKSVGVMGDARTYSYTVGIRAVTSIDGMTADWARIPWDVL
EKISVRIVNEVENVNRIVYDVTSKPPATIEWE
>lcl|NZ_CP026520.1_prot_WP_241688748.1_5401 [locus_tag=PC41400_RS27275] [protein=transglutaminase-like domain-containing protein] [protein_id=WP_241688748.1] [location=complement(6035144..6037669)] [gbkey=CDS]
MLTVIWYMLLLEWLSPLALGAAEFQPLGIAPYAVMFGYFLLISAGVRSFALLWTLKLAGIAAVSGYQFQKELFPFGPWPV
RLWETFLTDLDHMASGQLGALSGLTESLLFLFGAAAFISASHQLLLHRQNGAWFAAATGGYLLLLQVGLGIDTTPGLIRT
GALGLLLLAGLTVPRLRRQFGVSATGRGGSLPWMGISVLAAAVFYAAGTWAAGEAGSAAKMKPLSLSAVADKWAAFTRGH
KGSGTGQANGSSLRISGYGADDSVLGGALISDSRVVFTARTPVLAYWRGEAKSVYDGRGWSQPSGTLRPAFPALSGQAES
AEFSTDGALPAEAKTGSSAGNGQSAEGSITTSVVRKAPDAAAVKGPYTQEVTRISGQLGMQLFAGGEVLKLEEATNTSGQ
SVKPETFWLNNESGKVALPATDDPLKSYRVTVKPFSFGSGWSMPDPGPQASAFTPVSDAEIRSGLRSSVNGTGSVGERSG
TADSSEVNAAADSRTSAAGGNGSRSETVVGAEAGNPERNTQVPAAPAEPEPGELAGYLQLPEGLPQRVRDLTERVTAGSG
QDRLAAALALERYLRTRYRYSMDKPTLPAAGEDFVDHFLFVDKLGYCDHFSSAMVVMLRAAGIPARWVKGFAPGEAGERG
PDGSLAVTVRSRDAHSWAEAYIPGAGWLAFEPTPGFAAPSGAGAEAVPAAVSSPEEETAAADTASPWAAQLLPALARAAD
SPAARGSAVLLALLGAAWLLRRLVPRRASQAPLRRSAALRRGTGSTALMERLWLKVFRSHGRLAPQQTLREYVAGLAGLD
PGRRRALEEFAVLYEEVRYDRSGAGIIGKGRILELWKRIEG
>lcl|NZ_CP026520.1_prot_WP_042234977.1_5402 [locus_tag=PC41400_RS27280] [protein=DUF58 domain-containing protein] [protein_id=WP_042234977.1] [location=complement(6037791..6039629)] [gbkey=CDS]
MMKSWLNLAGTVAGAVGAHVLYLAAGGLAPLIIECLLWANAVSGASVLFFTLNGAEVSRRLGAPRIFAGRNAEVKLEVTQ
RSLLPVCWMVVREYWVQEGQGVSSCKSLLCFPWTASTVSLSYSLDDVGRGQYVLERTELLCGDLFGLFTKRRVVKAEAAF
TVYPAPESFGLRGVPGAESESRGKPSRLLRTTGVSAGVRAYAAGDPLSWIDWKATARRGSLQTQVRSGAVKPSRIIVLDT
DPGCGGVDTGCECVHSAGKGGEGRKSARSAKKGRNGSEDRSGISRKSDRSRENAWGEKSGSPAWTLVPNSANRNYLDGTD
EHPRANRRLAFGKSRHIRGKHAASGPCGLGQAMLEKRIRLAAYAVLEPDTGTGGIVTGTSFRERAVLAQSAQPYSFPGFQ
GASSFDPDDGAPVPFANASLELLAAVRGETGKGVHMLRKEHASQGPGVTLLYLTSCLDEALIVCAMELLAARRPLQIVYV
SQSHMLTSQEQKRLSVLREAGVETACFAAAEDRSAEAPVAGYGQRAENSGSDVNKRTFFIKKHKAKRMKVPASAEYSAGI
QEMEKASRNTYSGNVSRSSGSDDFFNEHSSVPHSIAGDKEKPQEGGTEDAIG
>lcl|NZ_CP026520.1_prot_WP_042234975.1_5403 [locus_tag=PC41400_RS27285] [protein=MoxR family ATPase] [protein_id=WP_042234975.1] [location=complement(6039639..6040619)] [gbkey=CDS]
MISNRTFLFDQPHTLLQKVVDQVAKVVVGKRDAIEKSVIAMLCGGHVLLEDVPGVGKTLLVRSLAATIGCGCKRIQCTPD
LMPSDITGVSVFQPQTAAFEFRPGPIMNPVVLADELNRALPRTQAAFLEAMEERSVTVDGVTYELPKPFLLLATQNPLDH
EGTYPIPEAQLDRFLMKLSLGYPEHGDEVGMLEQQLQRRSPDSLRPVLLQEEWVQLQNEAATVNVDESLREYIVSLAGAT
RRHPDLRLGASPRASAALMRAAQARALMQGRRYVVPDDIKCLAVPVFAHRLLPSTDALIGGRTPQAAVEAAVASVPIPVL
RFAAVR
>lcl|NZ_CP026520.1_prot_WP_042234973.1_5404 [locus_tag=PC41400_RS27290] [protein=polysaccharide deacetylase] [protein_id=WP_042234973.1] [location=complement(6040682..6042061)] [gbkey=CDS]
MLKTKFSHFRFTRIALILLLIIVVFPSIQPAHRAYAANADSGPAESIYEALISGDKVASDRTYAAPDKPTVYLTFDDGPS
KYTPEVLDILRKEEIPATFFVVGEHVEENPKLVKRIVSEGHALGNHSYDHTYKKLYSDYKGFWKQVEQTDRAIEKAAGFR
TGLMRAPGGTAENFDAFYFYYMKEAGYTLYDWDIDSEDSKRTGVPAAEIIRSIRKAPLKHELTVLFHDGTGHKETVKALP
EVIRYFRDKGYAFAKLDDRVKPAQFPVRKAGRWHRSAPGEAEFARTAAAMRDWAASASVSGGRTPVPSGEETAVGAGSVR
HRQTEPRLPLTLIVDGRRTVLPPELALFREERLEVPLRVLAQQIGARVSWDQAALTASVDYGFTRLDYNLRDYTLSVHLW
ARITAVYHLPNMRLLEGSLYVPLRKTVELLGGDILDYHADKSGREVTISGGTALFSSKL
>lcl|NZ_CP026520.1_prot_WP_042234971.1_5405 [locus_tag=PC41400_RS27295] [protein=nucleotidyltransferase domain-containing protein] [protein_id=WP_042234971.1] [location=complement(6042211..6042981)] [gbkey=CDS]
MEAVIREKLKEIEENEQVKVLFAVESGSRAWGFPSKDSDYDVRFVYVRRPEWYLSIDDKRDVIEVPINDMLDINGWDIRK
ALKLFRKSNPPLMEWLISDIVYQEAYGFKEEMLRLRSEVFSPKASLHHYLSMAKGNFREYLQRDQVRIKKYFYVLRPILA
CKWIEENDTNPPILFQELVDRLLPESGLRDQINELLRRKIAGDELDLEARVEEIHAFIGKEIEHLTQVAAASKAELADPT
PLLDDVYRRYLTLVWK
>lcl|NZ_CP026520.1_prot_WP_042234970.1_5406 [locus_tag=PC41400_RS27300] [protein=Gfo/Idh/MocA family oxidoreductase] [protein_id=WP_042234970.1] [location=6043155..6044315] [gbkey=CDS]
MGDIKVGVIGTGFSAMAHIEALQRVPSVQVVAVASSRLSHAEEAAARYRIPMYYGDAMELIHDPDIDVVHNCTPNHLHYT
FNKAALLAGKHVLSEKPLSVTVEESDELQKLAREKGLVAGVAFIYRHYPLVEEIKQQIASRTYGKPHYVLGGYLQDWLLY
ETDYNWRMDSRLNGPSRAVADIGSHWCDTVQYVLGTRIVSVFADLGIVHPVRFKPKDEQATFTNAAPKPEDCERVEVDTE
DFGSILVRFEDGTKGVFCISQVSPGRKNKLYFEISAEKGALRWDQEEANRLWIGSRDVPNQIQMRDASLMSSQAASITHY
PGGHEEGWPDALKNLLLDYYSLIGKRKARKTDETSSKFATFNQACGVMRIVEAVLQSHERQRWVDV
>lcl|NZ_CP026520.1_prot_WP_042234968.1_5407 [gene=bcp] [locus_tag=PC41400_RS27305] [protein=thioredoxin-dependent thiol peroxidase] [protein_id=WP_042234968.1] [location=complement(6044378..6044842)] [gbkey=CDS]
MSINTGQQAPDFELPANNGQQVKLSDFKEKKVVLYFYPKDMTPACTTQACDFRDKHPEFESLNAVILGVSTDSLKRHEKF
IDKYSLPFLLLADEDHAVSEIYGVWKEKSMYGKTFMGIERSTFLIDEEGVLVKEWRKVKVKGHIEEALTYLKEH
>lcl|NZ_CP026520.1_prot_WP_042234966.1_5408 [locus_tag=PC41400_RS27310] [protein=hypothetical protein] [protein_id=WP_042234966.1] [location=complement(6045012..6045749)] [gbkey=CDS]
MNKTRSKLMVSTLAAALLLGGGAALSSQVFAADSTDSGQTSTQQSAPSKHDGRDRKSFGQFGGPGSYLKDAATVLGLEEK
ALITELKAGKSLAQLAEAKGISQADLVKKLVEASTKEIDAKVTEGKIDSTKAAELKANLEKRIEKAVTVTGMQNFKGGKH
GREGGFVNPEKLAAILGITKEELKTQQEAGKTVAEIAAAKGITREQLLEKLKTELTPSLEKFIDAKKGDKPQKKDQKQAQ
TEQAE
>lcl|NZ_CP026520.1_prot_WP_042234964.1_5409 [locus_tag=PC41400_RS27315] [protein=ABC-2 family transporter protein] [protein_id=WP_042234964.1] [location=complement(6045902..6046687)] [gbkey=CDS]
MFYLSLILDYLKNYLKTRLTYRSDFWIEVVSDLLFNGLNLVFILVVFQQTALLGDWNRDQILFIYGFFMIPYGLFSTIFN
LWNFSERYIVKGEMDRILTRPAHSLWQVMLENMDPASLFSALAGLVVMIYSWVQLDLPVHWSDPLVMLMMVVSGVLIYGG
IYVSLSALAFFSDSPTGIMPLIWNIQNYGRYPVTIYNKLLRGILTWILPFAFVGFYPAAYFLDPLNWKGVALLTPVVGIV
FITIGLSVWNIGVKRYRGAGS
>lcl|NZ_CP026520.1_prot_WP_042234961.1_5410 [locus_tag=PC41400_RS27320] [protein=ABC-2 family transporter protein] [protein_id=WP_042234961.1] [location=complement(6046691..6047485)] [gbkey=CDS]
MYKAYVELIRIRFITMLAYRVNYYSGILVYALNIGSYYFLWKAIYGSQETLGGQTLGQMTTYIVVSWMSRAFYFNNLDRE
IANEIRDGSVAIQFIRPYNYVVYKMMQGLGEGLFRLLLFTTPGLVIVSLLFPVELPTDISLWLVYLLMLLFAFLINSQIN
IMTGLSAFFLENSEGMMRMKRVAVDLFSGLILPITMFPGWTAKLLEWLPFQAITYLPGSVITGKMTGSAVWQAVSIQVIW
FVALLVPIAFMWHKARTRLFVQGG
>lcl|NZ_CP026520.1_prot_WP_042234959.1_5411 [locus_tag=PC41400_RS27325] [protein=ABC transporter ATP-binding protein] [protein_id=WP_042234959.1] [location=complement(6047478..6048560)] [gbkey=CDS]
MPAIEVKDLRKEFRVQKSRSGLKGAFRDLFDREYNQVTAVKDISFEIPEGEICGYIGENGAGKSTTIKMLTGILVPTSGH
LQVNGYIPYKDREAFVKGIGVVFGQRSQLWWDIGVIESFQLLKKVYRVPENDFKARLDELSDRLQLGELLSRPVRKLSLG
QRMRCELAASLLHNPAIVFLDEPTIGLDIVVKTEIREFLKSLNQRYGTTILLTTHDLQDIEALCSRVIMLDDGRIIYDGG
LDQLKETWGKGKEVVLQFNDPVPLENLKLLTEGLDVTWEQDSALDARVLIPQRSANVSDVLARIVGVVQIADIRILETNT
DDIVREIYKSGSAEARKPSVPEAGKPGSGAEEDKGAVPHV
>lcl|NZ_CP026520.1_prot_WP_042234990.1_5412 [locus_tag=PC41400_RS27330] [protein=LCP family protein] [protein_id=WP_042234990.1] [location=6048878..6049918] [gbkey=CDS]
MLFVGILFALICTAGYFAYSIYSFADNINNKDTIQGFDKKYQGDDYKPPVWEGTGRVNVLLLGADSRGVENGEPPRSDTL
MIASIDPVTKKASLMSILRDTWVKIPGNDEGRINSAFTIGGPNLAMKTVSELTGIPIQYYLYTDFKGFIALVDAVGGIDL
EVEKDMKYTDAADGHVFDINLKKGMQHLDGQTALQYVRFRHDAQSDYARTERQRKFLTAVADKMQSTQSILKLPSILKSM
EPYINTNMSFGDMVKLGGLAFDARAKGIQSIQLPPENLVVERRINGAQVLSVDKTKLKRYIDDVFEGMTDAPAAGTVTGT
GTGSGSGSSSSSGTAGTSGTSKNTRQ
>lcl|NZ_CP026520.1_prot_WP_042234957.1_5413 [locus_tag=PC41400_RS27335] [protein=glutamate-1-semialdehyde 2,1-aminomutase] [protein_id=WP_042234957.1] [location=6050043..6051353] [gbkey=CDS]
MNHTRSEQLHEEALQHIVGGVNSPSRSFKAVGGGAPVFMKRAEGAYFWDEDDNRYIDYLAAYGPIITGHAHPHVTEAIIR
AARSGTLYGTPTELEITFARMLKEAIPSLDKVRFVNSGTEAVMTTIRVARAYTGRTKIIKFAGCYHGHSDLVLVAAGSGP
STLGTPDSAGVPASIAQEVITVPFNDIPALEEALTRWGDDIAAVMVEPIVGNFGMVMPKPGFLEGLCAAAHANGSLVIYD
EVITAFRFHYGTAQTYAGLNSRPEEIEPDLTALGKIIGGGLPIGAYGGRKEVMEQVAPLGPAYQAGTMAGNPASISAGIA
CLEVLRQPGTYERLEQLGAKLADGLRESASRHGVALTVNRIAGAISTHFCDHPVTNYDEAQDTDGERFGRFFRLMLERGI
NLAPSKYEAWFLTTAHTDEDVARTLQAADEAFRLLG
>lcl|NZ_CP026520.1_prot_WP_042234955.1_5414 [locus_tag=PC41400_RS27340] [protein=sensor histidine kinase] [protein_id=WP_042234955.1] [location=complement(6051702..6053468)] [gbkey=CDS]
MKPEERRRFVPFGYKLMLSYALCILVPVIIVGSFAVSTLVKSVREQAESSVQGTMRQIRDNIQYKIEDMRRLTDMLYGDT
TLAKHLKHYDEGWVSYEATTGYLLPKMMQTIQSSSLRLWLSVYLHNETLREVYYNHDNIDPLMTVGKLYELYHMKRIKDA
DWYKSFAPERYGTTMQWRQIEDDAKFSRLSLLRRIVDTTNGVELREIGFIRVSAHLQDLFESVDYTKVGEGSAIYIVGPQ
GDILYSSAADPNMTGKTWNESWTRNHLVLRQPLEGLDWSVVSLVPTAIMEKDADKVRMLTAIVCLISVLVIFFVGWMFSR
FFSKRVTKIVSVLHAFREGEFRKRMAYEGRDEFAQIAAALNDMGQHTEGLIREVYLTNLQKKEAELESLQAQINPHFLYN
TLSSISRLARFGDIDKLHRMVIGLAKFYRLTLNDGRTFIPLSKELEQVQAYVDIQSIKYGDRMEVLFDIQPEVNRYDTVK
LIVQPFIENALQHAWCGDHIHIRISGEIIGGDIHLKIIDDGNGIPQDRFRKLFDPLDEVESGYGIRNVDQRIKLHFGKKY
GVSLASRPGIGTAVLLRIPTYKHKKNRE
>lcl|NZ_CP026520.1_prot_WP_042234953.1_5415 [locus_tag=PC41400_RS27345] [protein=response regulator] [protein_id=WP_042234953.1] [location=complement(6053517..6055088)] [gbkey=CDS]
MYNVLLADDEILDLEGMQTFIPWERLGLHVVGAVNNGFDAVRVMEQEKVDILVTDVRMPSMTGLELARKSLERHTGLRVI
FLSGHQDFGYVKEALSLKACSYVLKPMDDNELIQSLETIIGDLNREQKRRKTERAFEQMRPIVHNEYVFRLLEGTAGRQE
LDDLLTEEGLEPIRYPIRAAVLETDDLSWKLNPYTAKERQDMLNDFGGQIMDLCRPRGITHMCRITKQRLGLLVTADEKG
IKELQEVIAYVKEHFPFTITIGLGNPAEDPSALQESYREALAALDAKMFYGKAKIIETREIPQEEIADAKQLEFRLDALF
NAMVQYDLVGIHDEIDRLFNVAVCLRSRFTIHNFATYLIVKLDGYLQTRNEQLFTLLGMELQNLDILLQFETIEDIRSWM
RRMLFEISEKLNMGNQSKNRKLIRDMMEYIKERLHENVTLRDVADRFSFSPNYLGHVFKEETGQNFSDYVIALRMEKAKE
LLGDPTLKIYEVADRIGYRYLPYFSRQFKETFGMTPMQFRRNH
>lcl|NZ_CP026520.1_prot_WP_042234951.1_5416 [locus_tag=PC41400_RS27350] [protein=extracellular solute-binding protein] [protein_id=WP_042234951.1] [location=complement(6055182..6056822)] [gbkey=CDS]
MAKWKKGVSLTLITVLSSIAVLSGCSSDSKPNATDTGKQGTAQTTGVKFGEQQLDISFYGNYDWFTMPVWGQDLATKWVK
ENKKVNVTGVPNGGNSEQKFNTLMASKELPDVIWLERGAQVEKLRAAGMLVSFDEYLDKYPNLKKWAGEATLNMLRSSDG
KLYQFPNWYTQRPAGNAGYMINLKIYKELGEPKLETYDDLYAYLKQVKAKYPEVTPFDSGVQAQNVEIMYPGFANDHPVT
NINMRAVPEGNELKSIFADPVYRESLTYSSKLFREKLITQDVLTQTKDQIDEKVNNGKVAVMAASSVTEFGSKGTTALQS
KDPSHPGYQMIWPIHKEGVDKTKVFPGHYATLGWNVNVITKSAKNPEAIFAFLDWMTGEEGQRVMFWGPEGEYWQGTDDK
GTPKFTDKYYSDATGRAKLMDSLVNLQWVGNTTYIDTAKTDIELKLPAEKQSWETRNQTQVAWKTVFNRTEFVNLDPPSN
TEEGMVQQRIKDIFTESRAKALFAKSDAEVTSILDKAEKDAQAAGYAKLLKYQTDRWQENKKKIGK
>lcl|NZ_CP026520.1_prot_WP_042234949.1_5417 [locus_tag=PC41400_RS27355] [protein=carbohydrate ABC transporter permease] [protein_id=WP_042234949.1] [location=complement(6056913..6057809)] [gbkey=CDS]
MLRANFGDRFMTVVIYTLLIFLAFITLYPFWNSVVISFNSGKDTMLGGITFWPREFTLENYNVVFEDKRIIQGFLITVSR
TVIGTVVGILATSLFAYGLSRRGLIFRKYYMVMAVVTMYFTGGLIPNFLLIRGLGLMDTFWVMILPVMINVWNMIIFMTF
FRSIPAGLEEAAKLDGCGTWSVFFRIVLPLSGPVIATLGLFTAVQHWNEWFLPSIYLNNTDLYPVQTLLRQILNSNIMSE
QLANLDSAALTQLSKMQSITTKSLSMATMVVSTLPVMLIYPFVQKYFSKGVMIGSLKE
>lcl|NZ_CP026520.1_prot_WP_042234947.1_5418 [locus_tag=PC41400_RS27360] [protein=ABC transporter permease subunit] [protein_id=WP_042234947.1] [location=complement(6057835..6058803)] [gbkey=CDS]
MAQVAVETSVSKEAKSPSIWKRFFKQWDIQLMVIPGIVFLLIFHYIPMYGVLTAFKDYDLFNGFFGSPWVGLKHFKMFFE
SPDFPRVMRNTIMISLFKLLIAFPAPIILALMLNEVRNKLFKNVVQTVTYLPHFLSWVVVAGFVTSFLSTDNGSLNIAMQ
KLSMIDEPINFLAIPEYFWTILISTGIWKDIGFGSIVYLAAIAGIDPALYESASIDGASRFKQIYLITLPCIFPVIIIFL
ILQIGSLMNAGANGGYEDILLLSANPVLREYSDVLDTFVYRTGIQNQLFSYATAVGLFKAVMSIVLLSGANLLARRTGQS
LW
>lcl|NZ_CP026520.1_prot_WP_042234945.1_5419 [gene=bioB] [locus_tag=PC41400_RS27365] [protein=biotin synthase BioB] [protein_id=WP_042234945.1] [location=complement(6059005..6060024)] [gbkey=CDS]
MHTEQQTDGRTETLSRWDHYADKALRGDKLEMEEALTVLRADDDELLPLMQAAFRVRKHYYGRKVKLNLIINAKSGLCPE
DCGYCSQSIVSKAPITKYPLLEKDVLIEGARKALEMKVGTYCIVASGRGPTNRELDQVIEAVQTIKGEMPLKICACLGIL
SDEQAVRLKEAGVDRYNHNLNTSASHFAKITSTHTYDDRVRTVDQAKRAGMSPCSGIIAGMGESEEELAEMAFALRELDA
DSIPVNFLNPIPGTPLAEQRSLNPRQCLKILALMRLICPSKEIRIAGGREVNLKSLQPLGLYAANSIFLGDYLTTEGQEA
GDDHRMLEDLGFEIELCAL
>lcl|NZ_CP026520.1_prot_WP_042234943.1_5420 [locus_tag=PC41400_RS27370] [protein=sugar phosphate isomerase/epimerase] [protein_id=WP_042234943.1] [location=complement(6060295..6061053)] [gbkey=CDS]
MTKIGIGVQLYTLRSETEKDFRGSLKKVAELGYEGVEFAGYGGLSPRELSELLKELNLTSVGSHVGIHRLQENLDEEIEM
NVAIGSKYVACPGWFPPERTEEGFREVVEILKESSRRFAEHGIAFGYHNHSFEFEEKIGDQTMFDAFFSALPENELFVEL
DVCWAHNAGLDPIGVMKQYAGRTPLLHVKDLRREEGKLLTVELGTGEVDLKPVLEAAEEIGTKWFIVEQDECQNPPFESI
AASREWLRQNYR
>lcl|NZ_CP026520.1_prot_WP_042234941.1_5421 [locus_tag=PC41400_RS27375] [protein=AraC family transcriptional regulator] [protein_id=WP_042234941.1] [location=6061224..6062117] [gbkey=CDS]
MTGIQDPASAVKDEVHILSAGYSVHRKPFVSNQRDGIQYYLLRLQTEGTCQVWQEGRYVTAEPGDLLLYRPGDAYELRIG
SQHETLPPDADTVYSGDYYLMCSGSWMDAWWERFKRPGIIRVGLDDSWLNIWREIFRERRRLHDNLQEIGDYLVRVLCLS
LDRVIGDLADLGSFPQKPRFSYMVQRMKTYIEQNASEAFTLQDLADHVGLSTSRAGHLFRETFGQSVMDYAIEVRISMAC
ERIRFGAMTLEQAAESCGFHSYSYFHRTFRARMGLSPREYRDGKLGSLEEPQVHGVT
>lcl|NZ_CP026520.1_prot_WP_129112544.1_5422 [locus_tag=PC41400_RS27380] [protein=hypothetical protein] [protein_id=WP_129112544.1] [location=6062222..6062428] [gbkey=CDS]
MRRAARCGSGDADAATASALLRRWRSHSASAATHSRFGCAVPAKTAAKQKQPLSSPGQGLFFLLRRIF
>lcl|NZ_CP026520.1_prot_WP_241688749.1_5423 [locus_tag=PC41400_RS27385] [protein=hypothetical protein] [protein_id=WP_241688749.1] [location=complement(6062586..6063209)] [gbkey=CDS]
MNRKKWLIGLILLCLAAGIYAVIRFNIDPTLAPEDVKLRARIVPAAEKPAEVPSQASAAYATVVEVELESTGGKALKQEG
YSYKVYPYVQNGTVAAWEPAPDALGKGQKPESYTFGPDAVTMPRALEMIERLTGASTNEEEAKKAGMSGGFVYSGETFPA
KVRYYAKEAGAGSGADANDGRTYILFSYHEKKWGKDVSWVKAVKVTP
>lcl|NZ_CP026520.1_prot_WP_042234939.1_5424 [locus_tag=PC41400_RS27390] [protein=inositol monophosphatase family protein] [protein_id=WP_042234939.1] [location=complement(6063374..6064234)] [gbkey=CDS]
MTTVNQEPYRVGGKSFTAVAINTAAKAGEWIKSRVGEIRRVDVKYSSHDLVTEVDKGSEKMIRNLILTHFPHHAFLGEEG
VEPGPEASAAALENVSDAEYLWIVDPIDGTTNFIHGFPFFSVSIALAYKGEVIVGVVYDPMRDELFVAEKGKGAYLKAAK
LGVSGESALSESLVGTGFPADREGALPINLAGINALMPKVRNLRNAGSAALHLAYVAAGRLSGFYEIGLNAWDIAAGALL
VQESGGRVTDTAGQPYTLGVRNVMATNGLIHEELQNELQQAGATGL
>lcl|NZ_CP026520.1_prot_WP_042234937.1_5425 [locus_tag=PC41400_RS27395] [protein=peptidase S24] [protein_id=WP_042234937.1] [location=complement(6064379..6064873)] [gbkey=CDS]
MGGEAHSRLGSGGFGAEALVVRRALELKGRVEIPSRGTSMRPLIREGSVCAFVPVAAADLRRGDIVLFEGEEGGLIAHRL
IRRGEEGPQAPLLCKGDANIGFDPLVPAAKVLARLETISRDGRVSSRLSPGFRIWGRIMTAFPALSRGFRLASRVARRLR
GQRN
>lcl|NZ_CP026520.1_prot_WP_042234935.1_5426 [locus_tag=PC41400_RS27400] [protein=hypothetical protein] [protein_id=WP_042234935.1] [location=complement(6064873..6065706)] [gbkey=CDS]
MTRFITRIGNERIGIETDSPVLGEWLRESFDRGFGDDGEEPRLRLNIKGGFGVPFVDFDVAAEKRGGKVIYTRADYRIEA
DEDLRAASVAVQDEFALKHALMNVYSAFLIRNGLGLLMHSSCAVQDGRGYLFAGPSGAGKSTAALLSRPRVLLSDEATVI
ALPEAGPGDGAESLALAHNSPFRSDSRWDIGEGIPAAPLEGMYVLRQSEEVRTEPMKAADALPIVMSQVFFWTTGAEETR
AVMQLCMRFLERVPVQELFFQKNDLFWDVIAPRKAGV
>lcl|NZ_CP026520.1_prot_WP_009676877.1_5427 [gene=fabI] [locus_tag=PC41400_RS27405] [protein=enoyl-ACP reductase FabI] [protein_id=WP_009676877.1] [location=complement(6065914..6066690)] [gbkey=CDS]
MTTSGLLNGKNILVMGVANDRSIAWAIAQSLAAQGARLAFTYESERVEERVRKLAGNIPDSLILPCNVTVDEDLVRLADT
LKGEFGVLHGVVHSIAFARTEELEGMFVDTSRDGYALAQDISAFSLVAVSKHMYPLMTEGGSIMTMTYLGAERVMRNYNV
MGVAKAALEASVRYLAADLGQHGIRVNGISAGPIRTLAAKGIKDFNSILHQVEEKAPLRKTTETAEVGDTAMFLMSHLSR
GITGEIIHVDGGYNIVGA
>lcl|NZ_CP026520.1_prot_WP_042234932.1_5428 [gene=uvsE] [locus_tag=PC41400_RS27410] [protein=UV DNA damage repair endonuclease UvsE] [protein_id=WP_042234932.1] [location=complement(6066911..6067912)] [gbkey=CDS]
MIVRLGYVAMSTEVKNASPSKTMTATLFSQLNDREAAVRKLERIGAENLHNTLRLLRHNRAYDIKVYRYSSKLIPLLGHD
MLEGWDPIPALAAEFKEVGDYAKLHGMRTSFHPDHFTVLSTPRKDVLLKSYADLERHAAMLEAMGLGEDAKLNIHVGGTY
GDKPKAAKRFVAHFADLNESARRRLTLENDDKTFTARETLEICEKVGVPMVLDVHHHQVNNEGEDILELWPRIQATWKGQ
LCVSLPDSADQPLPPKIHVSSPKNDKDIRSHADYVDAGPLLEFFRGIAGITPRLDVMIEAKMKDAALFRLMKDLEPEKGV
DILTEASVQINVP
>lcl|NZ_CP026520.1_prot_WP_042234930.1_5429 [locus_tag=PC41400_RS27415] [protein=hypothetical protein] [protein_id=WP_042234930.1] [location=complement(6068126..6068449)] [gbkey=CDS]
MGFQTEFNSVCKFKSEQELYELLEYGRCKMRKSGFRVYPTGQKVIAYSPANEAVAIVKISASIAEITFQGDEVTLVEMDL
VRKLTEEEARVQTALADEMFFGEQGSK
>lcl|NZ_CP026520.1_prot_WP_042234928.1_5430 [locus_tag=PC41400_RS27420] [protein=D-alanine--D-alanine ligase] [protein_id=WP_042234928.1] [location=complement(6068613..6069752)] [gbkey=CDS]
MSRKLRIGLIYGGKSGEHEVSLQTAHAVIGALDVSKYEITPFYITKQGEWRRGAALNGRVERKELLAFDASEGEAAGAAL
APFFSGLTSQAGGVLAEVAAAGEAAREEAVDVVLPLLHGTFGEDGTIQGLLEMANIPYVGAGVLASAVGMDKVAMKKMFA
QEGLPQCVYRHFTRSQWEKDQPFFLMETEISIGYPCFIKPANLGSSVGISKARNRQELIEAVEYAFQFDRKVIVEEFVDA
REIEVAILGNEEPLASVPGEIVSSSEFYDYKAKYVDGKSTMVIPAELPEETAAQIRELAIQAFQSIDGSGLSRVDFFLTR
TDGQIYINEINTMPGFTPFSMYPLLWKETGKPYAELLDDLIRLALERHTAKQQIKYTVE
>lcl|NZ_CP026520.1_prot_WP_042234927.1_5431 [locus_tag=PC41400_RS27425] [protein=amidase domain-containing protein] [protein_id=WP_042234927.1] [location=complement(6069785..6070807)] [gbkey=CDS]
MSWKTALYDYLHARNTAETEASAGVLANVVEDERFNRETDARYRRLIESRKERGVKALRGEMRLKLQHVREHEEGVTADV
ECGRQFFYEQGGRERKECIVIRERVGLIRKGSRWRIAQISARESEREKTKQPYSWLAEEGIYTPEGEISSSQPYLNYSIL
NHVGTYPETRKKLYDRARAVKYADTWWNGYNPAFIEFEVDCTSYVSQCLFAGGSPMNYTGKRGAGWWYSGKQGSRELWSY
SWAVAQALQLYVPASRRGLTGTLVSSAAELRPGDFISYDWDGDGRFQHNTFVTALDAGGMPLVNAHTVSSKHRYWDYKDS
YAWTDKTRYAFVRVSDEMDG
>lcl|NZ_CP026520.1_prot_WP_042234925.1_5432 [gene=acnA] [locus_tag=PC41400_RS27430] [protein=aconitate hydratase AcnA] [protein_id=WP_042234925.1] [location=6071076..6073790] [gbkey=CDS]
MPQKDAFNVRKQLTVGGKSYHYYSLQDLQQQFAGISKLPFSIKVLLEAAIRQYDGRAITDEHVKQIATWGDENADHNKEI
PFIPARIVLQDFTGVPVVVDLAAMRDTVARAGGDPKRINPLVPVDLVIDHSVMVDAYGNGMALETNMELEFERNEERYRF
LRWAQTAFDNFRAVPPATGIVHQVNLEYLASVAATKTVDGETTVYPDSLVGTDSHTTMINGLGIVGWGVGGIEAEAGMLG
QPLYFVTPEVIGFKLTGRLAEGATATDLALTVTQTLRKKGVVGKFVEFFGPGLEDLVLSDRATVANMAPEYGATVGFFPV
DNSTLEYLRNTGRSEEQIALVEAYYKAQGMFRTKDMEDPIYSDVIEIDLSTVVPSLAGPKRPQDRVELTAMKESFNDILR
TPVDKGGYGLSDEKIAEEVEVKYADGRTTKMSTGAVVIAAITSCTNTSNPNVMLGAGLLAKKAVERGLKTPAYVKTSLTP
GSLVVTDYLEKANLLKPLEELGFFVAGYGCATCIGNSGPLPDEVSQAVADNDMTVAAVLSGNRNFEGRVHAQVKANYLAS
PPLVVAYALAGNVNIDLTKDPIGYGTNNEPVYLKDIWPSNKEIEEAVATAVTAEAFREKYKDVFRANERWNQIAVPEGEL
YEWDKNSTYIQEPPFFQNLGQSLDDIKDVRGAKTLALLADSVTTDHISPAGNIKVDSPAGKYLIEHGVDKKDFNSYGSRR
GNHEVMMRGTFANIRIRNQVAPGTEGGVTTYLPTGEVMSIYDASMNYQDKNTSLVVIAGKEYGTGSSRDWAAKGTLLLGV
KAVIAESFERIHRSNLVGMGVLPLQFLEGQGWKTLGLTGRETIEISGLSNDVQPGEQITVTATREDGTSFDFKAIVRLDS
MVDVDYYRNGGILQTVLRQMMYNA
>lcl|NZ_CP026520.1_prot_WP_084706689.1_5433 [locus_tag=PC41400_RS27435] [protein=DUF1272 domain-containing protein] [protein_id=WP_084706689.1] [location=complement(6073938..6074141)] [gbkey=CDS]
MGLAMKKDCQICGHTLDAQSEAYICVHECTFCKNCTEERGHICPNCSGELVRRPRTPLTAAAACPLG
>lcl|NZ_CP026520.1_prot_WP_084706685.1_5434 [locus_tag=PC41400_RS27440] [protein=cation diffusion facilitator family transporter] [protein_id=WP_084706685.1] [location=complement(6074409..6075347)] [gbkey=CDS]
MEQRKSMIAIWISLISNVVLTGIKLVVGWLFGSQVLIADGIHNAGDVIATATAYSSMRISNRPADHDHPYGHGKAEVLGS
GAVALLLGAAAIYMGYHAVMALLEPPGEAHVLALIAAIISLVWKQWLYVYTMRLGRAVNSKGLIATAYDHLADVYASIAA
SAGIALALAGDYFDVPFLAYGDPAAGIIVSFLVIKLAYTMGIEAVNILMETNVEDVKLQQYKEIIESFPDVKRIDRLRAR
EHGHYILVDARIGIMGTLTIQQGHDIGRSIKEAIRAKHPEVDEVLVHLNPWYPDEELNPDAGPGITHVKSGG
>lcl|NZ_CP026520.1_prot_WP_255246869.1_5435 [locus_tag=PC41400_RS30645] [protein=hypothetical protein] [protein_id=WP_255246869.1] [location=complement(6075583..6075705)] [gbkey=CDS]
MSTSTGEGPGGYRRTAVQQDNRLPEAFFSRNAPCPASGAD
>lcl|NZ_CP026520.1_prot_WP_042234923.1_5436 [locus_tag=PC41400_RS27445] [protein=IclR family transcriptional regulator] [protein_id=WP_042234923.1] [location=complement(6075782..6076522)] [gbkey=CDS]
MEEAKSTVRAVERALDVLLCYTDATELTLTEISSKVSLHKSTVYRLLASLEAKGFLVRESDGDKYRLGFSIWELAAHMSQ
SDDPGMILLPEMERLRDQVNETISLYVRDGRERVRVQAVQSNHAIRRVAPIGARMPLFVGASSKILLAFGEAEVQEQAML
EAELAGILREPFLQQLHETRKAGYATSVEEREPGAAALAAPVFNRAGKLVAALAVSGPSNRLTLEAMLEQAPVVIEAARR
MGTMLR
>lcl|NZ_CP026520.1_prot_WP_042234921.1_5437 [locus_tag=PC41400_RS27450] [protein=alpha/beta hydrolase] [protein_id=WP_042234921.1] [location=6076821..6077834] [gbkey=CDS]
MSMNPINSTPLQPAPLDHIPVKSISRRKKVVLLSLLCFFVLCSSLILGFHGFIAWTLARPTIAPLSSNPAKAVGLPYQNV
TFPSSNGSSTLSGWYIPTAQASSQTVIFSHGYGGNREELWVPIYSLAKKLHDSGYNVLMFDYGYVHPENDRIVTAGIQES
KELLGAVNYIKEKGSERVYVWGFSMGAGTALQAALQTKDIDGMILDSTFLLNPDTLYHNMKQYVNIPKFPSLPLVRVFFP
LLNGVNMNQIPYQAVTTHEYPMPIYFIHGDEDERAPHEIAEGVYKNQHNVFSNYWMLPTGKHELIYSAKPDEYITRTFGF
LKTISRPQLPESGFVEM
>lcl|NZ_CP026520.1_prot_WP_042234920.1_5438 [locus_tag=PC41400_RS27455] [protein=Fe-Mn family superoxide dismutase] [protein_id=WP_042234920.1] [location=6077903..6079690] [gbkey=CDS]
MLSVYGSLMPLRILEEIRFWKTQEKEHTVVIRELVPTLEPVYVQALKEWEEVFAKTEAAAVQWVEWAIRTQNPNDPFLQL
QLRQLIDVSTKQSVQFIEFLRIIEKESKPVSSNPVVKTVIEHIIRESEYFLGTLKGLETAQKGSPYGTGGWGFGVPGHWP
PGAHSGLDGYAGQAGDGGLGTFGAPGAPGAPGAPGAPGAPGAWGAPGYYGPGGIILPGAGFGGGPGYSLGGAGSPSTSIG
GAGSLLEGVSTPMTAGGGTGPGQYGLYSVPVPGHDSDGPAPVLPGYSSGSTGYAVPQTGAGPGTGVPPGFGVGAPGPPAS
YTPGPSAYPGAGIGPVGPQGPWDTRTEGDRPPLSDMAAQSGVAPPPAHASGHTSGQGQTAPALSPYKPVPIGGHKLPPLP
YAYNALEPHIDEATMREHHDKHHKAYVDDLNKAEQMLAKARDSGDFALVKHWERELAFNGAGHYLHTIFWNIMKPNGGGK
AYGPIAAEINKSFGSYDRFKKQFTEAAKKVEGGGWAILVWSPRSHRLEILQAEKHQNLSQWDVIPLLVLDVWEHAYYLKH
KSNRAKYIEDWWKVVNWDHVNERFEPASKLTWLPY
>lcl|NZ_CP026520.1_prot_WP_042234918.1_5439 [locus_tag=PC41400_RS27460] [protein=DeoR/GlpR family DNA-binding transcription regulator] [protein_id=WP_042234918.1] [location=complement(6079893..6080678)] [gbkey=CDS]
MSLVGEERKEHILHLLNLHNKVRTHELVEHFEVSSETIRRYLEELEQENKLKRVYGGAVKFDLMREEPSPFNREVTKAEE
KKRIGRAAAAMVRDNDVIVIDDGTTTLQMINYLANTRNLTVLTPSVPALSLLIEYKNKDMFSGDIYFIGGKINAKHYRAS
GSVAESMMESFYVDKAFIAVDGLHPQSGLTSYDAERGLLVRKFMEHAAQTVALMDHTKNGKTHFYKISELKEIQAVICDA
EHPPEWNGDFEAIDTKWIVAE
>lcl|NZ_CP026520.1_prot_WP_042234912.1_5440 [locus_tag=PC41400_RS27465] [protein=saccharopine dehydrogenase C-terminal domain-containing protein] [protein_id=WP_042234912.1] [location=complement(6080869..6082053)] [gbkey=CDS]
MKVFCLGGAGRICREAILDLVQYSGFEQITVADFNEEEGRKVVRELHDPRVDFIKANVYDKEATVAQMRGYDIVMDGTAI
TLNGLSTACIAEAGCHGINLNGFGDEDASAALFERHGKTCVPGFGMTPGVTQMMAMHAASQLDTVDTVRVSHGSYRPIAF
SRSITETTTYEYDPNLPGRVVYENGAFIQVPPFARPREIRLPEPYGTSVQYIIPHAETKTLAKALAGKGVRLIEVRGTWP
QPNMRLVQALYEYGFMRNDRIRVNGSEVGIMDCISEYLYHSREGSETELYGYALHIEVTGVRNGVPMRLTLTHTHPASDG
SVRGWEGLRAYTRNVGIPMAIATVMIAEGRVNGAGILTPEEAFEPQAVFAELEKRGIRIHQDTEVLTEREPVQA
>lcl|NZ_CP026520.1_prot_WP_129112583.1_5441 [gene=phnW] [locus_tag=PC41400_RS27470] [protein=2-aminoethylphosphonate--pyruvate transaminase] [protein_id=WP_129112583.1] [location=complement(6082124..6083251)] [gbkey=CDS]
MPDNEYLLLTPGPLTTTKRVKAAMLRDWCTWDRDYNGLVQDIRARLVTLGAGRAEGYTCVLMQGSGTFCVESVIGSVIPQ
GGKLAVLTNGAYGDRIAQMARVLGIETAVLDSGELSPVSPGELEELLRRDEAITHVAVVHSETTTGMLNPIAEVGAAAKK
YGKTFIVDAMSSFGGIEMDMSGLHIDFLISSANKCIQGVPGFGFVLARTEELKRCEGQARSLSLDLYDQWDTMERQSGKW
RFTSPTHVVRAFHEALKELEDEGGIKARFERYTENQRTLSEGMEAIGFRALLDPSVQSPFITSFYYPDSPDFTFLAFYEE
LKKNGFVIYPGKISAAQTFRIGSIGDIHPADMRRLVRIIADVTGGTAATAAATRN
>lcl|NZ_CP026520.1_prot_WP_042234910.1_5442 [locus_tag=PC41400_RS27475] [protein=phosphonoacetaldehyde hydrolase] [protein_id=WP_042234910.1] [location=complement(6083295..6084143)] [gbkey=CDS]
MIKAVIMDWAGTTVDYGCFAPLNVFIEIFRRKGIEITQKEAREPMGLLKWDHIREITKMPRVAGLWEQKFGSRPNDRDVD
ALYADFEPLLFASLRNHCEPVPGVLELMGRLRERGIKIGSTTGYTEEMMDIVVEEAGRRGYRPDVLVTPTEMPAGRPYPW
MIYENCIRLGVYPLHHVVKVGDTLSDIREGLNAGVWTVGVVLGSSELGMSEAEVSACPPEELKRRCEEVRGRFYEAGAHY
VIESVGKLDAVLELIEASLKAGAGVSAGAAASASVPQGARHV
>lcl|NZ_CP026520.1_prot_WP_042234908.1_5443 [locus_tag=PC41400_RS27480] [protein=putative 2-aminoethylphosphonate ABC transporter permease subunit] [protein_id=WP_042234908.1] [location=complement(6084270..6086030)] [gbkey=CDS]
MMIVRRVQSAARDLRSGLTGTRGRIRRRHGKEERMQQALIALMIAVLAVTVLLPLIGLFSKAFYDKTGQFAGLENFVVYF
TTPALLRSLLHTLNVSLLTTVISVGLAFLYAYALTRTDIKGRTLFRYIALLPLFAPTMMHGIGLIYLFGNQGLVTTGLFG
LLPAGLDIGLYGPVGIVLAEVAYTFPQAMMIMLVSLAVTDNRLYEAAETLGAGKWRTFLSVTLPSVKYGVISALFVCFTM
SFTDFGAPKIVGGNYNVLATDIYKQVIGQQNMTMGAAVGILLTLPAVLAFAVDRFVERRQKTLVSSKSVPYRIQKGSWRN
AVFYVICSLIGGAILLLLGAVIFASVIKVWPYELSFTWNHFDFTSVAGEGLAPFWNSMQAALYTALIGTAVTFVFAYLIE
KVRHWHALRQFAYFLSILPLALPGMVIGLAFIFFFNRPDNPLSGIYGTIWIVVLANIVHFYAVPFITATAALKKLDKEFE
TVSESMRVPFYRTFARVTVPMSLPAVLEMFVYYFVNSMVTVSAVVFLYSADFKLASVAIVNMDDAGDTAPAAAMSVLILL
TNIVVRIGYEQLTKVMRNRASAWQRK
>lcl|NZ_CP026520.1_prot_WP_042234906.1_5444 [locus_tag=PC41400_RS27485] [protein=putative 2-aminoethylphosphonate ABC transporter ATP-binding protein] [protein_id=WP_042234906.1] [location=complement(6086049..6087095)] [gbkey=CDS]
MNQAHLSIRGIRKTFGSFTALDSIHLDVRKNEFVCLLGPSGCGKTTLLRIIAGLEHPDTGTVSAGGRDITALPPAQRNFG
MMFQSYALFPNMTAAQNIAFGLKSRKLGKRDVAEKVRESLDLVDLWEMRDKYPSQLSGGQQQRIALARAVALSPDFLLLD
EPLSALDAKVRHKLRGEIRALQEKIGVTTIMVTHDQEEALTMADRIVVMNNAEVAQIGTPREVYERPASPFVADFIGAIN
FLDGAELQERGVRGAGGMMAVRPEHIGAEEPGSRGGIPAVVRGIEFRGPFYHVMLDVLGSGGEPLGRPMTIHVSAQHYQQ
LGLEKNKKLSLRLPEERLIAFEPQAAAK
>lcl|NZ_CP026520.1_prot_WP_042234904.1_5445 [locus_tag=PC41400_RS27490] [protein=putative 2-aminoethylphosphonate ABC transporter substrate-binding protein] [protein_id=WP_042234904.1] [location=complement(6087169..6088275)] [gbkey=CDS]
MKKTASLCLTLLMAISLLAGCGKSGSEAGAATKPPEKAGGNEGTSQNAALKGDLTVYTALEDDQIKAYLESFKKTNPDVK
VNIVRDSTGIITAKLLAEKDNPQADVVWGLAATSLLVLDQNGMVEGYSPKGIERVSPMFKSTGTPAKWVGIDAWETAFIV
NTKELEKKKLPVPKTYQDLIKPEYKGMIVMPNPASSGTGFFDVTGWQQLYGKGPSWEYMDKLHDNIAVYTHSGSKPAKMA
AAGEYPIGISFGYRGIQEKKKGAPVEVVFPQEGSGWDVEANALMKKATIKPEAKAFLDWAIGDEAMKAYNKNYAIISVND
GSGTIPEGYSKDPLSQLIKYDLNDAAKNRDSILKEWEKRYNAKSEPKS
>lcl|NZ_CP026520.1_prot_WP_042234902.1_5446 [gene=folE] [locus_tag=PC41400_RS27495] [protein=GTP cyclohydrolase I FolE] [protein_id=WP_042234902.1] [location=complement(6088563..6089150)] [gbkey=CDS]
MPAKEYKNTKVADNREEIEGHIREILRLIGENPEREGLLDTPARVARMYEEIFAGYEADFHDVLGVTFDEQHEELVIIKD
IVYYSQCEHHMAPFFGKIHVGYIPSGKIAGLSKFARLVEAITRKLQVQERITSQIANTLDEVLEPHGVMVVVEGEHLCMC
SRGVKKPGSKTVTSAVRGSFRTDSALRAEFLSLIK
>lcl|NZ_CP026520.1_prot_WP_042234900.1_5447 [locus_tag=PC41400_RS27500] [protein=TetR/AcrR family transcriptional regulator] [protein_id=WP_042234900.1] [location=complement(6089140..6089748)] [gbkey=CDS]
MPSNKDKIIAAAMELFHEQGFQATGLEEILSRCGVVKSNFYYHFKSKEELGRQVIQQKIALMEEQIFGPSLGNRTLGPKE
RLGRLFRLMQVSAEEHDCRRGCYIGNLVVELGDRYEELRRLLVNFLEDIEHRIEACLQEGAANQEINLHGMKARETAVAL
TALLQGGMLLGKAHRSSFALQYGGKLVKHIIGNGWEGEQHAG
>lcl|NZ_CP026520.1_prot_WP_042234898.1_5448 [locus_tag=PC41400_RS27505] [protein=YneF family protein] [protein_id=WP_042234898.1] [location=complement(6089842..6090084)] [gbkey=CDS]
MWQIIIPILTLIGGLVGGFFIGVFYLRKQLESMQNNPEMIQKMAKQMGYNMNKQQMNKVQTMMKNQNLTNKKPNKNARSK
>lcl|NZ_CP026520.1_prot_WP_042234896.1_5449 [locus_tag=PC41400_RS27510] [protein=HD domain-containing protein] [protein_id=WP_042234896.1] [location=complement(6090163..6091194)] [gbkey=CDS]
MKYVPIDSIEPGQYLGKTIFTASGAVMLAEGVQLTVYMINTLRRIGVSMLYIQDQETSDIEIPEVVSEETKRIVIRQMLD
TFDSIRSGKDFNTRHMSKNVDTLLDEILKNKDVLIQLTDIRTKDNDMYLHALNVCMISTLIGINMNLNPQQLKELAIGAL
LHDVGKLDAITDDESKDDRLHHTWRGFEALKAKREYSLLIAHVAFQHHETPDGLGEPRRLLGEQIHLYAKIVSAANTYDN
LLQGSGLNAGLLPHVAIEHMMAMAGTKLDRDILIHFLRTVSVYPTGISVRLSTRETGVVVGQHRGLPGRPVVRIIKQDGG
KEYDVKEMDLAKHTTLFIEHVLA
>lcl|NZ_CP026520.1_prot_WP_042234893.1_5450 [locus_tag=PC41400_RS27515] [protein=methylated-DNA--[protein]-cysteine S-methyltransferase] [protein_id=WP_042234893.1] [location=complement(6091191..6091790)] [gbkey=CDS]
MDDSGRGTVATTGEPVLIETDTLDDSPIGPLLLARTERGLCAIEFGNGPEAVRKLEHWTDRWIPGGTVVPSPAGLREIRE
QLEEYFSGQRRSFDLAFDLRGTEFQLRVWKALLSVPYGATASYKDIAAAIGSPQAVRAVGGANNRNPVPIVIPCHRIIGA
GGQLVGYGGGLSVKTALLELEGWSPADVRHTSKNREWKA
>lcl|NZ_CP026520.1_prot_WP_042234891.1_5451 [gene=queG] [locus_tag=PC41400_RS27520] [protein=tRNA epoxyqueuosine(34) reductase QueG] [protein_id=WP_042234891.1] [location=complement(6091864..6093111)] [gbkey=CDS]
MRITLADASERDLLLLKQEIIQAAPSLGIDKIGFASAEPFHELKDILLRHREKGYESGFEEPDIDKRVHPDRSFEGVKSI
LAIAVAYPSKLVNPPRSEAGAYRGILARAAWGDDYHLVLRERLKRLGAFLEERVEGVRFMDMVDTGALVDRAVAERAGIG
WIGKNCAVITPEWGSWVYLGELLTNLPFPADTPIMDQCGECRICIDACPTGALVGPGQLNSSRCISFVTQTKGMIPDEFK
EKIGNRLYGCDTCQIVCPVNKGKNWTHHEDLQPDHEVVKPLLIPLLTMGNREFKERFGRSSAAWRGRKPIQRNAVLALGN
FKDRSAVPVLASLLKDDPRPELRATAAWSLGKIGGEAALEAVLRAQRSERDEAVLAELDNAARKLGGLTPSGGDAAAVEA
GGPQADGAGAAPAAE
>lcl|NZ_CP026520.1_prot_WP_042234889.1_5452 [gene=lepB] [locus_tag=PC41400_RS27525] [protein=signal peptidase I] [protein_id=WP_042234889.1] [location=complement(6093259..6093867)] [gbkey=CDS]
MTDSGREPRSSSVPAPETGGVEGGGRISGRRNWSRLGTFVVIILAAAVIQRYGFNISAVRGISMEPTLHEGQRLFVNKTA
AWTGSFSRGDVVVLKEPDGTGTEKIHPYLVKRIVAAAGDRVEITDGRVFVNGEELKEPYTDERIEDGDYGPFTVGKGYCF
VMGDNRRAGASLDSRSFGPIAVRSLVGRAEWIVWPPDALKGL
>lcl|NZ_CP026520.1_prot_WP_042234887.1_5453 [locus_tag=PC41400_RS27530] [protein=DUF402 domain-containing protein] [protein_id=WP_042234887.1] [location=complement(6093869..6094420)] [gbkey=CDS]
MESFEHFVIKSFKHDGHLHRMWLENWKVPQHELHPEHQRENMNVFINNQTKIQEADGKEWTSRIPGVSFFIPGEWYNIVA
LLEEGGVRYYCNVASPFYRAGNVITYIDYDLDVIRMAGGDVHVVDREEYEHHKALYRYTPLVESKVKAGLDALVKRIGKQ
DAPFGDESVRRYYELWKDRETEA
>lcl|NZ_CP026520.1_prot_WP_042234884.1_5454 [locus_tag=PC41400_RS27535] [protein=GNAT family N-acetyltransferase] [protein_id=WP_042234884.1] [location=complement(6094701..6095132)] [gbkey=CDS]
MRVRSFRLSDYSLISGLLQEALSDPCYKETMQAFAKQLSWDGELVLVAEEQDELVGMVIGTIDDNKGYYYRLAVAPGHRR
KGIGKALIEGMKQRFMQRKVKRILVTVDHHNELVLPVYESAGYNHSDFTRTANRLSIVKKASM
>lcl|NZ_CP026520.1_prot_WP_042234880.1_5455 [locus_tag=PC41400_RS27540] [protein=6-phospho-beta-glucosidase] [protein_id=WP_042234880.1] [location=complement(6095558..6096880)] [gbkey=CDS]
MDKSLKIVVIGGGSSYTPEIVEGFILNYDKLPIREIWLVDIPQGEWKLNIVGSLAKRMVEKSGLPIQVHLTLDRREALPG
ADFVSTQMRVGMLEARAWDESIPLKYGVIGQETTGPGGMMKAFRTIPVLLDICRDMEELCPDAWLLNFTNPAGMVTEAIS
KHSSIKSIGLCNAPIGMHKWLMKEYDVPASSIYTEFVGLNHLHWITQITVDGESKLEELVGSGSGYSAKNVPQMEWNASF
IRGLGAVPSYYLKYFYLKDEMLEEQIEASKKDGTRAEVVKRVEDELFELYKDPDLAEKPKQLEKRGGAYYSEAAVNLMCS
LYNDTKDIQTLNVRNDGILDFLPNESSIEVNCVVTRQGPIPLSVTRVPEAVKGLIQAVKTYEILAIEAAVTGDRSIALQA
LAHHPLISSVNTAEKMLDEMLEQNRAYLPNFFPAAEHSNV
>lcl|NZ_CP026520.1_prot_WP_042234878.1_5456 [locus_tag=PC41400_RS27545] [protein=BadF/BadG/BcrA/BcrD ATPase family protein] [protein_id=WP_042234878.1] [location=complement(6096903..6097865)] [gbkey=CDS]
MTYYLGMDAGGSKTYAVIADETGRVVGTGRSGMGNHQIHHDRAAASIGGAVDEALREAGLTKADIAFAYFGLAGADREID
FRILRKLLEPIGFANWDLACDTIVAMRAGTTKPYGVVAICGTGVNCSGMNRQGEAYQCGGFTYMFGDFGGGGGLAVEAFR
SVVRAWDGRGEQTVLTGLVLEQLGYETVGDMFHDFLDHNKSVPLELAKLLFTAAGQGDQVALDILKEQGEEIGLSIKAIV
NKLGMAEEEFDVVLAGSVLMRGNGDYIEKYIVDAAKKAAPASTIRRLDAEPVVGSVLLAMERGGMTVSEEVSEKLKQLSM
>lcl|NZ_CP026520.1_prot_WP_042234876.1_5457 [locus_tag=PC41400_RS27550] [protein=LacI family DNA-binding transcriptional regulator] [protein_id=WP_042234876.1] [location=complement(6097919..6098947)] [gbkey=CDS]
MKISIFDVAKRAGVSVVTVSRVLNDSPTVRENNRQKVRQAIKELDYRPNSAARSLARGKTGVIGLTLTTLQDSLFDGIVK
CVNDALKEHGYFLALSIGPDTSDSDWGEGSNFLYQEDRVDGIIVLSPLKEENDVLELKRKKIPFVLVDNQLEDMSASVVN
VDNYMGGLEATRHLLSLGHTKIAHIAGPELYRSARERKRGYIDAMAEAGHPPFLLVDSDFDIRSGFDIAKSWLAEGDRPT
AVFAADDFIALGVIEAFRDAGLHVPRDLSVIGYDDQAFSSAIHPRLTTVRQPSEGMAQEAVKLLLKIVNGTQKRNASVKL
EPKLIVRESTASPDGRTLEIDR
>lcl|NZ_CP026520.1_prot_WP_042234875.1_5458 [locus_tag=PC41400_RS27555] [protein=sugar ABC transporter substrate-binding protein] [protein_id=WP_042234875.1] [location=complement(6099011..6100390)] [gbkey=CDS]
MWMTRKPYHLTLAAVLAASVTLAGCSGGDKKTEGTGGSGKSGDTVTITHYTQDAPDKAYVEELIPEFEKKFPNIKVKVVK
APYEQFESKLQSMVAAGNSPDVTSHWGDAGFAEYYNKGLVRDMTDLLKEDNFKASDYGISDHLMDIYKVNGKTYGIPVYS
YVSVLVYNKDMFDKAGIPYPPSDYEDKSWTFEKMQEYAKKLTKISKNLNDTQYGIDWGWSERDMRPLYFGASAYPEETFT
KNGGNPEVSKFNSPEVKDAYRTMFDMMHKDKSMIDPVTTKAVAGEGGDPFAAGKVAMSVGGAWLLAGSNDFGFKVGVAAV
PAGKNEKARDVLFVDPLFILKDSKHPKEALEWIKFQLSKETQEKAVELSGGTPPANSLAGEKYFSYFQNIDPKDLKNVYE
GGLKYGVESWNHMIPGYAELNTIIKNEMEPIENGKKSVDEVMPDLEKHINDVLAKRQKK
>lcl|NZ_CP026520.1_prot_WP_042234873.1_5459 [locus_tag=PC41400_RS27560] [protein=carbohydrate ABC transporter permease] [protein_id=WP_042234873.1] [location=complement(6100422..6101306)] [gbkey=CDS]
MSSTVPTQPARAYKPSRLKAGRRTWFDWFVFAMLIVLSVLMILPSVHMLSTAFKTRAESLRFPPTILPEKFVTENFKKLF
ELDFLNWYQNSLLIGIMTVIGTVLSSALVAYGFSRYRAKGKDALFAFLLATMMLPYPAVMIPQFILFKELGWIDTFLPLI
VPAFLGSAYFIFLLRQFYTSLSNELFEAAKVDGCSELRQWWSIALPLSGPALATVAIFSFIWSWNDLLGQVLYLNSTEKF
TLPIGMAGMVSSTTRPPSWNVVMAASLLALVPILVLFSVAQRYFVQSIVLTGVK
>lcl|NZ_CP026520.1_prot_WP_009676921.1_5460 [locus_tag=PC41400_RS27565] [protein=sugar ABC transporter permease] [protein_id=WP_009676921.1] [location=complement(6101303..6102196)] [gbkey=CDS]
MRTKTARHQTLWFYIFISPWLIGAVLFALYPIVSSAYYSFTEYDIVNPPKWVGFQNYVDLFHDDIFWISIWATVRYTLLS
VPIQLGLSLLFALLLNQKIPFKGFFRTVMYFPSMISGVAMALIWMWIFNPQSGLLNQGLSLLGLPGPLWLFDEKWAIFAL
VLMSLWTVGGGMVIFLAGLQSVPPSLYEAAELDGVTRLRSFWHITLPMISPVFLFQLIMGLIESFQVFTQAFIMTKGGPN
YSTMFYVYYLYQNAFKNFKMGYASAMAWLLLIVVMVLALLIMKGSKRFVHYEGGDGV
>lcl|NZ_CP026520.1_prot_WP_042234871.1_5461 [locus_tag=PC41400_RS27570] [protein=hypothetical protein] [protein_id=WP_042234871.1] [location=6102611..6102808] [gbkey=CDS]
MAQQERRHLHELSSLDASAWDEEELSYHHSVMSELSPWLNAQGTAIHAQVIREIEKRRENTVQPT
>lcl|NZ_CP026520.1_prot_WP_009676925.1_5462 [locus_tag=PC41400_RS27575] [protein=superoxide dismutase] [protein_id=WP_009676925.1] [location=6102894..6103508] [gbkey=CDS]
MAHQLPALPYPNNALEPHIDATTMEIHHDRHHNTYVTNLNAALEGHADLQNKSLEDLISDLNSIPEAIRTAVRNNGGGHA
NHSLFWETIAPNAGGQPTGALAQAIDSELGGFDKFKEDFAKAATTRFGSGWAWLALGKDGQLKVYSLPNQDSPLFEGDTP
LLGLDVWEHAYYLKYQNKRPDYISAFWNVVNWDAVNKRYEAAKK
>lcl|NZ_CP026520.1_prot_WP_129112584.1_5463 [gene=mutY] [locus_tag=PC41400_RS27580] [protein=A/G-specific adenine glycosylase] [protein_id=WP_129112584.1] [location=complement(6103637..6104977)] [gbkey=CDS]
MEYPTEQKRFFSANLLAWYLIHRRDLPWRRSRNPYYIWISEVMLQQTRVDTVIPYFHRFVDLFPTVEALAEAPEDEVLKA
WEGLGYYSRARNLQQAVREVHERYGGIVPDTKEDISSLRGVGPYTSGAILSIAYNKPEPAVDGNVMRVLSRYFLIEDDIA
KPAARVGMERLARDLIPEGSAGDFNQALMELGAMVCTPRSPQCLTCPVMDHCSGRLAGREETLPVKSKAKPPKPERRLVA
LIEGEGENAGRILIRQRPQEGLLARMWELPHVELRAAASGGARGSLAAAEPPGPAGGPGLLLAAADEAAAASAGADSPAE
AQPAAGSAAAGPEAPAKRGRRGKAAAAGSEAPAAWPGDAHSMEQLRIALAEEEHVDVLPQGWIADTNHTFSHLYWDMKVY
RCRLGGQGGAPAGQLPYRYRWMSPEELERYAFPNVFLRVLKMYAEE
>lcl|NZ_CP026520.1_prot_WP_042235814.1_5464 [locus_tag=PC41400_RS27585] [protein=hypothetical protein] [protein_id=WP_042235814.1] [location=6105274..6105828] [gbkey=CDS]
MKEARTAPGRPAVIPSGFRWLAPYLAVWGIQGIVAAAVDFFQEWIDLDWLAQGIFAAAVLASIVIWVKMRRKAGAAREGS
ISLVMLAFPAVMLIGAVAVLEYVQAVGPYFVALLRAFLLSVVFVLLGAAADRRLFWLGLWLFALTIVMALWYLGYSVVIL
QGMSGLSLLVCAWSLTRPLTPVTR
>lcl|NZ_CP026520.1_prot_WP_042235813.1_5465 [gene=acpS] [locus_tag=PC41400_RS27590] [protein=holo-ACP synthase] [protein_id=WP_042235813.1] [location=complement(6106278..6106664)] [gbkey=CDS]
MIIGVGMDMVDIARVAGLLREKSGDRFMRRILTEEEQQLAAGKRGRLAEFTAGRFAAKEAVVKALGCGIGRQAGFHDVSV
LPDEAGKPHCFLSEGARQRLGLGQDVVVHLSITHTDTMAAAYAVVERP
>lcl|NZ_CP026520.1_prot_WP_042235810.1_5466 [gene=nadE] [locus_tag=PC41400_RS27595] [protein=ammonia-dependent NAD(+) synthetase] [protein_id=WP_042235810.1] [location=complement(6106877..6107686)] [gbkey=CDS]
MSLQQEIIRELGVKPQIDVQEEIRSRVDFLKQYVKSAGVSGLLIAISGGIDSAVAAGLCKIATDELTKESGKEYKTVGVF
QPYGTQEDIHDSYATAEAFNLKYRMETNIEETVNEIALEVEFSLKNAGDSRHMSKEGKGNVKARTRMVVQYAIAFEMNLL
VVGTDHASEAITGFFTKWGDGAVDITPLSSLNKRQVRALAAELGVPDSVLHKAPTAGLWEGQTDEKELGVSYEDNSDYLE
GKEINPAAAEKLEKHYMKTRHKRGRIPGI
>lcl|NZ_CP026520.1_prot_WP_042235807.1_5467 [locus_tag=PC41400_RS27600] [protein=glycosyl hydrolase family 18 protein] [protein_id=WP_042235807.1] [location=complement(6107818..6110103)] [gbkey=CDS]
MKSNKRIKMTWLGFFALLLALLPLFNTGAGTASAAEPAKIVGYFTSWGIYGRNYQVKDIDGSKMTHLNYAFADICWGGVH
GNNSTDSPNKQTWSCTDSHVPLQSKSVPNGTIVLGEPWADVNTPYSGYSYEECDQKALCGNFAGLRDLKKKNPSLKTLIS
VGGWTWSNRFSDVAANAATRETFANSAVEFIRTYGFDGVDLDWEYPVAGGLSGNTYSPADKQNYTLLLKKVREKLDAAGT
ADGKKYLLTIASGASQKFANNTELSEIAKTVDWINIMTYDFHGGWEKSTNHNAALYPDPNDPSTGDIKKYNTSDAIDIYF
QSGVPANKLVLGLPFYGKGWKGCPPGPNNDGQYQTCVGGWDGNVLPTGTWDDWASGNSGTFDYGDIMANYVNKNGFTRYW
NDTTKTPYLFNPTSGTFISYEDTQSIAAKTAYIKSKGLAGAMFWETSSDCRTSPKFSCTIKLLDKVAADLMSPAVPDTQA
PTAVTNLVSTGKTSTSVTLSWTASTDNVGVAGYEVSYGTTKVNVPGTTANITGLTANTAYTFTVKAKDAAGNVSAPASVT
VTTDGGTTTPDTQAPTAVTNLVSTGKTSTSVALSWTAATDNIGVTGYDVTYGTKTVSTTATSLNVTDLTPSTAYTFTVKA
KDAAGNVSAPASVTVTTDAATNPGSPVQPTFTVTSDWGTGYNFSFSIKNTGTTPITNWKLEFDYTGSITSVWDASIVSSA
NNHFVIKGAGWNNTLQPGATVTFGGAGLVKAQPTNIVVTGS
>lcl|NZ_CP026520.1_prot_WP_042235804.1_5468 [locus_tag=PC41400_RS27605] [protein=BrxA/BrxB family bacilliredoxin] [protein_id=WP_042235804.1] [location=complement(6110573..6111007)] [gbkey=CDS]
MVMSFDSYMRDMVQPMREDLTRIGVKELLTPEQAAEALENSEGTLLLVINSVCGCAAGQCRPGVAKALQNEVKPDHLYTV
FAGQEKEATAKAREYLAPYPPSSPSIALFKDGQLVHFIQRHQIEDRSADMIAGDLTDAFEKYCK
>lcl|NZ_CP026520.1_prot_WP_255246872.1_5469 [locus_tag=PC41400_RS30650] [protein=hypothetical protein] [protein_id=WP_255246872.1] [location=complement(6111134..6111262)] [gbkey=CDS]
MAKSNNSKEQDIHEKENKQRKSTPSESGGVDKKLSGPNFPAT
>lcl|NZ_CP026520.1_prot_WP_009676348.1_5470 [locus_tag=PC41400_RS27610] [protein=PspA/IM30 family protein] [protein_id=WP_009676348.1] [location=complement(6111536..6112234)] [gbkey=CDS]
MGVFSRIKDMTKASIHEVLDKVEDPVVMLNQYIRDMEEEIAKAEVTVAKQIANERKLKQRLEESVRLSADREGRAAAALK
DGEEQAARQALEEKLYHDEKIAEYTDLHEASKGHASELTQQLHDMKDEYYKLRNKRNELVARTQLAKAKKQMAQVSYNGV
IESGNASRGFQRIEEKIIQLEVEAEIARKPYVPASGYSSVSSYNPSDSAKQAKVDEQLQQLKNKIGGTKAEK
>lcl|NZ_CP026520.1_prot_WP_042235801.1_5471 [locus_tag=PC41400_RS27615] [protein=PspC domain-containing protein] [protein_id=WP_042235801.1] [location=complement(6112237..6112752)] [gbkey=CDS]
MKKLYRSRTDKKVTGLCGGLAEVLNVDATLLRLLTVVAVFITSGSLILLYFIASFIVPKEPGFDPPRGPYGPGPYGGDSY
GPYGNNNNNNYGGSYGGNQAYNGWNGKEARNGRQQSAAGSSFGSPKADNIDEMMKDIEKKAMQREIEELRGKLDRLEKHQ
KPTNNDSKGDV
>lcl|NZ_CP026520.1_prot_WP_042235798.1_5472 [locus_tag=PC41400_RS27620] [protein=PspA/IM30 family protein] [protein_id=WP_042235798.1] [location=complement(6112756..6113418)] [gbkey=CDS]
MSILKRIRDITVATLNERLEESEDPIRLIDQYLSAKREQLMQSERLYQQCTAHASSLRQQYLSALETKEKREQQAALALK
AGEENMARLILHEKLQAEERSDQYRGLYEQAGQSAAELEEELQAMRADYQEVLDKRGYYMARMESARLQQRLNSRTGYGD
GSSSPRMFYRMEEHVSDMEMQAHVLRDLRRTTREGGSQNSNPALEKELEALRRKINREES
>lcl|NZ_CP026520.1_prot_WP_042235794.1_5473 [locus_tag=PC41400_RS27625] [protein=hypothetical protein] [protein_id=WP_042235794.1] [location=complement(6113454..6113747)] [gbkey=CDS]
MKRNPNMGLAIVLIAFGAWLILGKLGIHLGPIMSYLIPIGMVVVGYFGIQRGSKLGWVIALLGIFALLGKMTTLFIILIA
VAMIAFGVSMLKGTSNS
>lcl|NZ_CP026520.1_prot_WP_042235791.1_5474 [locus_tag=PC41400_RS27630] [protein=metallophosphoesterase] [protein_id=WP_042235791.1] [location=6113973..6115040] [gbkey=CDS]
MKHFQRAAAVLLACCTAVCLLPLGRQTIPAAEALLAAAPPQDSGQQTGAEAGADGGRTASPEETNRALLTFSVLSDIHIE
AHDKLSQDLLRKALLDHAALAPDSGMMVLNGDLTDGNPGDYNALAAVLNAMPHPPLHATMGNHEYYRMWRTPFGSVDTSR
LNPFWSTREAVGLFTRCFGYEKPYHAAEMAGYTFIFLSGEGYRDVVKDAREDAYLSAEQMNWLGGRLSEASRTHPGRPVF
VFLHQPLVHTLKGSGQERGVTDYKAFRRVLADYPQAVLFSGHSHLDWSLGGQAVRTDFLSVGSSSVRRVLNFDQKPVNPA
KSESLLVQVYPDKLVIRVREHYSRSWIGTYVHELR
>lcl|NZ_CP026520.1_prot_WP_053228804.1_5475 [locus_tag=PC41400_RS27635] [protein=spore germination protein] [protein_id=WP_053228804.1] [location=6115143..6116705] [gbkey=CDS]
MASSFTPFPRPGYSELYQERLSRSLPFNILRTRQVTGESADIQHREFRAGAAGQNGIKAALFYTEGLVENRAIDQFVLQP
IMMQMKSLDTETGHAADLMTLLEDYYIPASGVQRVENMGRVIELLNSGSTILYLDGVDTALAVDLTGWKERAIEEPSSQG
VIRGPKAGFTESIKTNLSLVRRRINDASLRYDGQKIGEITKTSIVVAYINGIVNEDVLAELKRRLSLIQTDSILESGYVE
EFIQETSWSPFPTLYSSERPDVVAAQLLEGRVAVFVDGTPFVLVAPALFGEFFHSPEDYYNRADISSLLRLLRFVSFLIA
LLGPSIYIAITTFHQEMLPTDLLFSLASQREGVPFPALLEAYLMEITFEIIREAGIRMPRAIGQAVSIVGTLVIGQAAVD
AGIVSAAMVIVVSITAISSFVIPAYNLSISVRILRFLFMLLAATFGLFGVSTGLILLALHLCHMSSFGVPYMAPFAPFQL
NEQKDTLIRMPQWLLNKRPRLLSRNTWRQPPPDKNPAPRK
>lcl|NZ_CP026520.1_prot_WP_042235788.1_5476 [locus_tag=PC41400_RS27640] [protein=Ger(x)C family spore germination protein] [protein_id=WP_042235788.1] [location=6116721..6117917] [gbkey=CDS]
MLKRLTGLLAPLLALCIVTGCWSKREMNELAIVVGIGVDKIDSQIKVTFQVVDPGEVAPKKSSGGRSPITVYESKGDTYL
EAMRKITTKSPRQMYPSHLRVLVIGESIAREGLSEILDIFTRDWEHRSDFIALVSKGSPAGEILKVLTPIDKIPANELYS
SLETAEKKWAPVSVVTLDKLINNMTETGKESVLSAVILKGPADKGQERNNIDQIKPETYLQYAGLSVFKKDKLLGWMNES
ESKGYNYITDNIHSTVHKAACPGGGIAVLDLIRSKTNVKARMKNGKPVILISLRSEENIGEMRCVIDLRKRENIRYLEKS
VADDIRRRMQDSIGRARDLGSDIFGFGKKVYQKYPAYWKQAKEEWTSVFRELPIEYDIQVNIRQLGRSNNSFINRLED
>lcl|NZ_CP026520.1_prot_WP_042235786.1_5477 [locus_tag=PC41400_RS27645] [protein=hypothetical protein] [protein_id=WP_042235786.1] [location=6117920..6118126] [gbkey=CDS]
MLAVTIIWLAGIALGAFQVPTLWKRRLLKDLAAFAVLLLSGLVLSTFLALSVPIPKPLDVVNFLIGRK
>lcl|NZ_CP026520.1_prot_WP_042235784.1_5478 [locus_tag=PC41400_RS27650] [protein=endospore germination permease] [protein_id=WP_042235784.1] [location=6118240..6119337] [gbkey=CDS]
MDRLNPRQLTMLATMYYIGSSILVSPAYLAERAGNDAWISGLLGLMIGFLPLLLIYALEKSYPGKHLGQIFESALGKLTG
RIIGFAYFLTYPMGIITLSLRDIGDFLTTSLLRDTPLEVLILFTLVVISIAVRLGIGTISRTAELFFPWVILLILLLLVF
VAPKADIENIEPVLASGWQPVMAASLSLAGFPFMEHCLFMTVFANVKGKSSLALQLGKGTLISGLLIVLVVFMCLAVLNP
GKTATYVFPTYELGKLIEFGQFVRRIEVIVAFIWFFTIFFRVSFVLYSMVTGLSYLLKLHDYRVITFPVAVISGALALLW
YPNITELNNFSKMWTYITLLSGLGLPILLLGGSAVQKMFRGPKPP
>lcl|NZ_CP026520.1_prot_WP_042235782.1_5479 [locus_tag=PC41400_RS27655] [protein=NAD(P)/FAD-dependent oxidoreductase] [protein_id=WP_042235782.1] [location=6119445..6120719] [gbkey=CDS]
MIYDVIIIGGGSAGLMAGVAASANGARVLLLDKGDKLGRKLGISGGGRCNVTNNKEMDELIRHIPGNGRFLYSAFSHFNN
QDIIAFFENLGIRLKEEDNGRMFPVSDKAKTVVDTLIRQVRSQGVTIRTNCSVDRVLYEDGRTTGVRLQNGETLHGKSVI
VASGGKSVPHTGSTGDGYAWAEAAGHTITELFPTEVPLTSAEPFIKSRELQGLSLRNAALTVWNAKGKKLITHQGDMIFT
HFGISGPISLRCSQFVVKELRKSQKGSVGLTIDLFPDQSTEEVYEHTLALAAAEPKRAIKNVLKGAVSERLIPLLLRNAG
LDENLTHANLPKQEWMELSKLLKAFLILINGTLSIEEAFVTGGGVNLKEIDPRTMESKLTEGLFFCGEVLDIHGYTGGYN
ITAAFATGYTAGKHAAMRAGEAAV
>lcl|NZ_CP026520.1_prot_WP_053228803.1_5480 [locus_tag=PC41400_RS30020] [protein=DUF3139 domain-containing protein] [protein_id=WP_053228803.1] [location=6120904..6121230] [gbkey=CDS]
MKKLLVIGGTGLVFLVLVILVVIRINERSLAKELKNYLIQEKRYREDDIEQVSSKFGKMPQYAVFVSFKDEPEVTYSYVK
KDKWIQLGPSDAEILSGKTFKHIDDRAH
>lcl|NZ_CP026520.1_prot_WP_042235779.1_5481 [locus_tag=PC41400_RS27665] [protein=hypothetical protein] [protein_id=WP_042235779.1] [location=complement(6121525..6122010)] [gbkey=CDS]
MSDTPIFFEFTQERDMRLAYDTLFELGYRPGVSASEGKPTLHIHVEGQDLASAMQIAQAHAGELIDAPPAGAVGCSEPEA
YAAAYDLDTIRIPAHLIQDPEEDASEGTLAAAGGGDGLGAAYDAYSGAAGAAGADGSYEEPQRQFDPSEDDLDQFSAGIH
L
>lcl|NZ_CP026520.1_prot_WP_241688923.1_5482 [locus_tag=PC41400_RS27670] [protein=ABC transporter permease] [protein_id=WP_241688923.1] [location=complement(6122108..6123322)] [gbkey=CDS]
MTPAKLFKRRLLDDWTYQYRILRTAVDWTVAVYLIVPALIFASIYYSAWLRTPPEWFGYIPASLVAVFLYLFAWSGTVRF
FIEEADQLFLVRSEGWMPAIKRRGLLYSLALQALTTALLAGALLPLLRAEGSGGALKLAALFLAVYAAKVNLGLLRQLAA
LAWAGWKRVAVSIALFAAGFLIYWACVQLILADTAAGYLAGPLLLAAAVPLGRRRLRAAGAFHADVQREREARLKLASLM
LRGIVEGKKPRQRRVPLLLGRSQRVVRGTGASAVLADSLVKSVLRRPGQWKLYAQLTAVLAFAAWRLPGPMHVVLWAVAA
FVLAYWMKSVARETLGASFFRMFAWSEINKQEALRKASFALAVPGLALIGLIAGGSISLPLGIAALVLSVPAAYAAAFLV
TIWY
>lcl|NZ_CP026520.1_prot_WP_042235773.1_5483 [locus_tag=PC41400_RS27675] [protein=ABC transporter ATP-binding protein] [protein_id=WP_042235773.1] [location=complement(6123342..6124040)] [gbkey=CDS]
MEKASIHIKKAGYGDSGAILRDIDFRILEGQLIGLIGPNGAGKSTTIKSMLGLLKDVEGNIEFADGRASVAYVPEHPVLY
EELTLWEHMELAAAAYGLERPVFEERAEALLAQFKLSADRHRLPVAFSKGMQQKVMLIIGFLNRPSLYIVDEPFIGLDPR
ATKDFLELLRKERERGAGILMSTHVLDTAERICDSFLLLSEGRIVAQGTLDEVRQESGLPDGTLFDCFHALV
>lcl|NZ_CP026520.1_prot_WP_042235770.1_5484 [locus_tag=PC41400_RS27680] [protein=membrane protein] [protein_id=WP_042235770.1] [location=6124173..6124487] [gbkey=CDS]
MHNHNSSDIDSNKSAAMSAYLIFCLPLMVARHSRFAMYHANQGLLLLLYAVFAHLLLVLLPGVENLLMPPLDISVFALAF
IGIRNAHDGRMKPLPLIGKLVLVR
>lcl|NZ_CP026520.1_prot_WP_042235768.1_5485 [locus_tag=PC41400_RS27685] [protein=aldolase catalytic domain-containing protein] [protein_id=WP_042235768.1] [location=6124686..6125639] [gbkey=CDS]
MKHDQHKILDCTIRDGGLVNNWDFSVEFVQDMYRGLNAAGVEYMEIGYKNSAKLLKSDNAGPWRFLNEDFLREVIPTKND
TKLSALVDIGRVDENDILPREQSQLDLIRVASYIKDVDKGLQLVQKFHDMGYETTLNIMALSHVMENELNEAFELINRSP
VDVVYVVDSYGSMNHKDIDYMVNKFQRLLPEKKLGLHMHNNMQLAFSNTVIGAEKGVEFLDTSVFGMGRAAGNCTTELLV
SYLKNPRYDVRPVLDLIERHMVAMREKVEWGYIIPYMVAGTLNEHPRVAMAVRNTADKDKYVEFYDKLTTPETIANN
>lcl|NZ_CP026520.1_prot_WP_042235765.1_5486 [locus_tag=PC41400_RS27690] [protein=hypothetical protein] [protein_id=WP_042235765.1] [location=6125807..6127903] [gbkey=CDS]
MSAKTKHDAFEEFYTRASNTGPAAGTFFNDKAYLFYTNGNLLDLICMETYLADDKQEKIRLAKSPGKKKGPLGIEYLTKV
AYGGDGVSGGYIPALAACAHEGVLYLFYADQRGTLYQMTSPDGTEWSSALTLSERMRLRTDSRFAAVSAGTRLYLIGSGA
DSKLALNWTDDGGQTWERTDAREQRPAKDGKTEEVGIADVTHTAACSYREPGGEVRIMVGYIQRSPDKKAVIPRKKEALK
TWLLHHEENAPPRKVNEPHSIAEDENVSHFAIVPGSIENGCTGSVLQMFVSGKAEKNIKARRYEYSASSGHWSPPILRNW
GHMSTPTGHHVDAFQFFKPTDEGHIRSEIWFTQTFDGSGMKKHLGIGRYRSDRLTLTDTLTNWANDDHLTFLGVVESPPP
HVLNGSELTDSICSFRFKWSKEEQAAVSAKYKFGASLNFSGKVPVVGLGGEMKIAYERTRSVTSEMKSIKEFSIELYPSA
TKKTATHLYIRPNITRHTYKLADWQENDLGVQLSLFSVSDLSIEARKGRSLSTFPKQPDPNRIENYVQRFTALPDYADKQ
LSFSMPVNWSESMKIEENFIKVSKTVTSVSNSVSVECEPAGEAPGDIFSIGVGSSFMYTYEVNVSTQFSSQMGCTVRYPN
PRPGETDDISEVKLLMYVFYPDKNKTDRCYWIPQETKNQFPWCVAWSVDELVRQGSSQ
>lcl|NZ_CP026520.1_prot_WP_053228802.1_5487 [locus_tag=PC41400_RS27695] [protein=hypothetical protein] [protein_id=WP_053228802.1] [location=6128093..6128506] [gbkey=CDS]
MKKTLVSSVLAVSIVAGALAAPSASFASALKTPPLANSVSTTVAPTAVQPDGLISNIIKKAITSALRYGGSYLGKLLEKL
SPSAGQWLTKYSGKIADFLDSITNWQENIIATGLVSLGIPPAEAWEIAKAIVWLAGI
>lcl|NZ_CP026520.1_prot_WP_042235763.1_5488 [locus_tag=PC41400_RS27700] [protein=hypothetical protein] [protein_id=WP_042235763.1] [location=6128603..6129154] [gbkey=CDS]
MNNYVLLFGLLLLLFFGLNVLLNYLSHRDHEPAPSWKTKIWAIPLLSLLIIGPVIGFAFLYMTFFRGIEHTSSLISFTGK
ADLFTFSLVILLSFLFFETFIHPLTHALIRYGLKRPPSIYSRQIITIIADCLLIYVFAHLIPGVYIKDFLSALTLSVALH
VIEWVLAGIMILYKKNNKKNLNM
>lcl|NZ_CP026520.1_prot_WP_009676335.1_5489 [gene=nrdI] [locus_tag=PC41400_RS27705] [protein=class Ib ribonucleoside-diphosphate reductase assembly flavoprotein NrdI] [protein_id=WP_009676335.1] [location=6129936..6130295] [gbkey=CDS]
MLVVYDSRTGNVQRFINKLNLKAVRIEEDLTIQEPFVLVTYTTGFGQIPGKVLDFLKHHSSLLRGVAASGNQNWGQSFAL
SADRISSAYGVPVLAKFELSGTSSDVERFHKGVRHLEAY
>lcl|NZ_CP026520.1_prot_WP_042235761.1_5490 [gene=nrdE] [locus_tag=PC41400_RS27710] [protein=class 1b ribonucleoside-diphosphate reductase subunit alpha] [protein_id=WP_042235761.1] [location=6130282..6132366] [gbkey=CDS]
MKHIELNNELMIPGEDGFYRLEKDREALDAFLEEVQGQTVPFGSIKERYDYLIGEDFYENVYEHYTLEQIEAIHETIRES
GFKFQSYMAASKFYKDYALRTNDKKQVLESYEDRAAIVALHLGEGDFGKAQRIAKAIIEQRVQPATPTFLNAGRSRRGEM
VSCFLLEMDDSLNSIQYVIGTCMQLSKIGGGVAVNMSKLRGRGEPIKGVEGAAKGIMPVMKLLEDAFSYADQMGQRRGSG
AVYYSVFGWDIIEFLDCKKINADEKSRIQTLSIGLIVPQKFYELAEQNKPLHVFAPFSVYKEYGRHLDDMNIDEMYEELL
ANPNVKKKVVMSARDMLTKIAVIQLESGYPYIMNRSNANRGHALKDIGKVKMSNLCTEIFQLQETTEIGDYGQGDIIRRD
ISCNLASLNIVNVMETKSIRDSVHSGIEALTSVSDNTTVANAPGVVKANKELHSVGLGAMNLHGYLSKNRIAYESPEAKE
FARAFFMAINFYSLEKSMEIARDRGITFADFERSEYAKGTYFAKYEEKDYRPESDKVSALFEGMDLPSPEDWKALKEQVM
KHGLYHAYRLAIAPTQSISYVQNATSSVMPIVDQIETRTYANSTTYYPMPYLSKENMFFYKSAYRMDPFKIIDLIAEIQE
HVDQGISTVLHVNSDISTRELARNYLYAAKKGLKSLYYTRTKRLTMEECLSCSV
>lcl|NZ_CP026520.1_prot_WP_042235758.1_5491 [gene=nrdF] [locus_tag=PC41400_RS27715] [protein=class 1b ribonucleoside-diphosphate reductase subunit beta] [protein_id=WP_042235758.1] [location=6132590..6133549] [gbkey=CDS]
MKAVNWNKPDDDYTWSFWQQNVMQFWTEEEIPLSDDKMVWIEMSPDEQDVYKKVLGGLTLLDTLQGGVGMPRLVEHMDGL
QRKAVLSFMGMMEQIHAKSYSSIFTTLASNEEIDGIFRWVEENGRLQQKADLINRYYNSIQSEKDLYLAMTASVYLESYL
FYSGFFYPLYLAGQGKMTNSGEIIDLILRDESIHGLYVGTLAQELFASLSEADQQEAAEEQIKLLLELYEIEKLYTEELY
TPLGLQDDVLRYVRYNANKALMNLGRDPYFPEEPVHPVVLNGIRTQTKQHDFFSKKGNGYVRALRVEPLSDDDFSFEGL
>lcl|NZ_CP026520.1_prot_WP_042235755.1_5492 [locus_tag=PC41400_RS27720] [protein=GNAT family N-acetyltransferase] [protein_id=WP_042235755.1] [location=complement(6133715..6134122)] [gbkey=CDS]
MTFKIRTDLPAKAAYYELFQSTGWDGDGIWTEDMLHEAVRNSWYVVTVYKGDRLVASGRMVSDGVIQCILCEMMVLPDYR
NRGLGSQIMDHLLDYCKSKGIRWVQLACAKGKQGFYERYGFRERPADAPGMGLFL
>lcl|NZ_CP026520.1_prot_WP_042235752.1_5493 [locus_tag=PC41400_RS27725] [protein=SDR family oxidoreductase] [protein_id=WP_042235752.1] [location=complement(6134501..6135403)] [gbkey=CDS]
MNVFVTGATGWIGSVTVDELLKAGHKVVGLVRSDVSAASLEAKGAKVLRGDLDDLDSLRRGASDAEAVIHLANKHDFANP
AESDRAERAAVETLAEALVGTNRPFMVATGLSGLYQGRPILETDPSPDVSPDSHRGGAENLALDYVDRGVRTMIIRFAPT
VHGAGDWGFVKFLVSAARKHGVSGYIDDGSHAWSAVHRSDAAKLIRLGIEKAPAGARLHAVAEETISTRAIAEAIGRALD
IPVVTIAPEDAVAHFGFVGSFFKLSMNGSSERTRELLEWAPAGPTLIDDILAGAYKEAGK
>lcl|NZ_CP026520.1_prot_WP_042235819.1_5494 [locus_tag=PC41400_RS27730] [protein=TetR/AcrR family transcriptional regulator] [protein_id=WP_042235819.1] [location=6135714..6136298] [gbkey=CDS]
MNKEHNPVGRPRNEQARTAILNAVDDLLFEVGYAAMTMKGISERAGVSRQTVYRWWSHKAEILFEASAADAEEELAISLS
GDPLDDITAYLEALLRFLAHSLAGAAYRALLGEAQHDSVVAELLSAKDPLGDSARVVIERALGKDTLPVPLEQATALLIG
PSFFWILSGRDPAQLVPRELAENFIHGLQARQSR
>lcl|NZ_CP026520.1_prot_WP_042235750.1_5495 [locus_tag=PC41400_RS27735] [protein=LysR family transcriptional regulator] [protein_id=WP_042235750.1] [location=complement(6136612..6137469)] [gbkey=CDS]
MEIRHLLTFKTVVEKGGFKKAAEYLGYAQSSITTHIKDLENEVGAPLFDRIGKKVILTHYGQQFLPYAIKIIELHTQALN
MDEEPIGNLKIGISELLTVQRIPSILLEYKKLYPKVNISLKSLDHCDISSNLQNGDIDMALILEKEDWPLPELHCEKLKR
EEMVLISSLENKEDLKTILYTEESCCYKSLFENYLESKHIEVEQILKFQSIEAIKQCVIGGLGCSLVPYFSVREEIEGNK
IKGERVSHELNANMSTYLVYHKDKWISPAMRSMITLIRTHVKTWS
>lcl|NZ_CP026520.1_prot_WP_042235746.1_5496 [locus_tag=PC41400_RS27740] [protein=SDR family NAD(P)-dependent oxidoreductase] [protein_id=WP_042235746.1] [location=6137635..6138372] [gbkey=CDS]
MNKNIIIVGAGQGVGLETARRFGKEGYSVGLIRRGQEQIEEMAAELSAEGIDTYFEPADAYDPKQIERAIKLLADKMSGI
DVLFYNVPGPLGKAYVPATEISTELITEFLNMRVVSALKSAQAALPYVKESKGSILYTSGQSDRYPFPYTAAIGAPQAAL
RLLSEHLHNELKGNGVFVGYFPLHNPPLYTDLEKEKERTDLPAGFHLDERVVASDVAEKIYQLNITRDRFECAVAASKNV
NENQG
>lcl|NZ_CP026520.1_prot_WP_042235744.1_5497 [locus_tag=PC41400_RS27745] [protein=TolC family protein] [protein_id=WP_042235744.1] [location=complement(6138546..6139919)] [gbkey=CDS]
MKKIIATLTASAFVFGMMPAAWAADTHSPYYEIKDGNLVQLHATTDKPLNKDEIRLPALEKVSKLTLDTVLEWTLNNSYN
MKLLYLKTAALGSKSSDLKNQSENLYGGSFEGYKLPASPQDYMTPEAMKKYQIDPTKATPDSFFWIGPVVETNTVLNKVI
EGQGQMVNGINQILQAQREQMKTTSHQLDTDMWNNLLQTDEAIEGIKLQVTAQYVQLLGSKKQIALMHEYKGVAEKEMKR
AQVLSDAGLASPEDVETARKAVNKIGDDIQTLENNYRLGLVQLCFDIGVEYTPDLLLEEPAAVSENVQPVVKKDTETLLA
NSFKLKMDHNGIDEAIWQQQNTVTKNTYGGQYQSVNTGIAYQKQEQTKAELVKKIYATYSDAENAYQIVLNEQRNLKDAQ
ADVAKMKIRYENGFVSKHDFDKFSFKLQQAQTTLEGARLKYFALVRKAEAMDRGFIS
>lcl|NZ_CP026520.1_prot_WP_042235741.1_5498 [locus_tag=PC41400_RS27750] [protein=ABC transporter permease] [protein_id=WP_042235741.1] [location=complement(6140097..6141323)] [gbkey=CDS]
MIPLPLKVGSVIAEEWRQIFKDRRLMAILFLVPILYTTLFGSIYIQHKVTEMQTVVFDEDNSQLSRQIVQAFDQSETFRV
VNAYHSEAEVTEAISRGEAKVGVIIPKEFEARIKHGDVQPVMTLIDGANMMISNSATRAANEIVTTYSYGVSGKKLQQQG
LQDEQVSSTFSAVPFRYRVLYNPTFNYSDFMIFGLAGAILQQVLLLGIALTITRDKEKGIWSRFAEWKKMPWRLAYAKSV
PYFIIGMLNAVVVFTIMLFGFDLPMRGQILMAMLLSACFNFAVLGIGYFISLFSSAQLGSTQIAMLIAVPSFMLSGYTWP
FEAMPKALSILGHLLPLTYFLEGVRHVFVKGNGYDVIVKDCIALVLTGLITYFAAFLLTRFAMFRKQTDEPASAGLPAVQ
QESTSLTV
>lcl|NZ_CP026520.1_prot_WP_042235738.1_5499 [locus_tag=PC41400_RS27755] [protein=HlyD family efflux transporter periplasmic adaptor subunit] [protein_id=WP_042235738.1] [location=complement(6141304..6142593)] [gbkey=CDS]
MKFVRPVVYVVMAGLIGGGTFLLSSKEALSSTQNRPANLPTAYIEANQVAASFKAGGRVTEILVKEGDTVKKGQVLARIQ
SQEIEAKVAQAKAAVALSQGKIAEAKGAEATAQAKKQQGQEAVTLTAETAEKQVKQAEAVVKAAQTKVDALKNGARPEEK
KQAEIQLNATKEVFQTAEESYNRVKKMKEEGLVAQTDVDKAKVSYEEAKAKYEVAQQQYALAQQGPRVEEIQAAEAQVEQ
AKAAVELAKANMDQVKIRQGDVNAAQASIQQAQGAVKSAQSGQQQAQAAQSEAETYLSYTELLAPSDGIIISQSAQLGEM
VGSGFPVFTMETSADTKWARFYLKETDAAGLKTGDTVTVKVTANGAAVTGKVSTVAPAADFAVKKATQNASDTDIRSFGV
KVELSQLPESVLTGMTVQWEGKAGDTVAP
>lcl|NZ_CP026520.1_prot_WP_042235735.1_5500 [locus_tag=PC41400_RS27760] [protein=TetR/AcrR family transcriptional regulator] [protein_id=WP_042235735.1] [location=complement(6142660..6143229)] [gbkey=CDS]
MDQRRENILDAAKKSFALFGYKGTTVDQIAKIASVGKGTLYLYFETKEDLLNEIVKSLIAEMRKLADASISKGGTIMEKL
HSATYSILVYRKEHELLLKLSQEVAQFGTPASQHALREIEEGVIGYIKQHLEIAVANGSIKPCDPEITAFVLVRMYLVLL
VDWERKHPPLTNEQISDLIQLYMMDGIAL
>lcl|NZ_CP026520.1_prot_WP_042235733.1_5501 [locus_tag=PC41400_RS27765] [protein=response regulator transcription factor] [protein_id=WP_042235733.1] [location=6143462..6144163] [gbkey=CDS]
MSSKILLVEDESKIARVLQLELQHEGYEVSLAEDGRKGLDMALAEPWDLILLDIMLPELSGLEVLRRLRQQSAVPVILLT
ARDAVPDRVSGLDQGANDYVTKPFAIEELLARIRNLLRLTTQQSQAENDQILQAEDILMNLKSREVTRMGKSVELTPKEF
DLLQYLLQHKGEVKSREDIISDVWGYDFVGDTNIVDVYIRYLRQKLEKNFPGKLIHTIRGVGYLIKGSGPDES
>lcl|NZ_CP026520.1_prot_WP_042235729.1_5502 [locus_tag=PC41400_RS27770] [protein=HAMP domain-containing sensor histidine kinase] [protein_id=WP_042235729.1] [location=6144153..6145520] [gbkey=CDS]
MSLKLRFALLQTSLLFLILLGVNFFIYYLFIENNSRNETQNIERKAAKLLLQENVVQHPEYWENPFLLKDFLVANELIRL
IGPDSTVQKEIYTDETLLQRPAKFIPATGTVSKLVSTDNGFCLYVQVPVMENERQIASLEIARKLVTLNDNVQVLFLALF
YTTLGALVLSVIIGYIYASLVFNPIHRLAQTMESIERSGTFRRLEIQSTRKKDEIAQLGQTFNEMIGRLEQHFEQQKQFL
ADASHELRTPLTIIESYARLLKRWASNDPQIREEAIEAILSESVRLREMTKSLLTLADSDERQRTEWQEFELRPLVQATA
ASIQLTFDREVRLDLPEEDTPLKMKGDPEKIRQLLIILLDNAIKYSQLPINIIVREQPRHLQISVVDKGIGIEKDDIPQL
FNRFYRVDKARSRKNGGVGLGLAIAENIVKLHAGTLKIQSEPDIGTEVHVRLPKD
>lcl|NZ_CP026520.1_prot_WP_042235728.1_5503 [locus_tag=PC41400_RS27775] [protein=Hsp20/alpha crystallin family protein] [protein_id=WP_042235728.1] [location=complement(6145768..6146265)] [gbkey=CDS]
MNQPDMNKWLKWEHVEKFLDQPQKYIENIKWIEGFVRDVFDRTMPPPQMNADIQQTQPRSQQKGKQQTSSQQPRESVNGD
TFETHTDVFVRLKIPKSEDPRSLQVMVKTSHVKILGLTDEPRKLIKLPSPVLPKTAKATYKEGVLQIQVRRRKYKDNYHE
AYIEY
>lcl|NZ_CP026520.1_prot_WP_042235725.1_5504 [locus_tag=PC41400_RS27780] [protein=spore germination protein] [protein_id=WP_042235725.1] [location=6146420..6146647] [gbkey=CDS]
MPAIVGNVKILTVGPSSVVQFGDCLVISPNSSSKSYAGAGSFNTGDLPHVFNAVSATNTNDSDAIEAEANKTATA
>lcl|NZ_CP026520.1_prot_PC41400_RS30495_5505 [locus_tag=PC41400_RS30495] [protein=spore germination protein GerPB] [pseudo=true] [partial=3'] [location=6146644..>6146778] [gbkey=CDS]
MNWTIHQTITIHQLRVDAVSNSSVLQVGSAGSIRALSNQYNTGGF
>lcl|NZ_CP026520.1_prot_WP_042235997.1_5506 [gene=gerPC] [locus_tag=PC41400_RS27790] [protein=spore germination protein GerPC] [protein_id=WP_042235997.1] [location=6147163..6147861] [gbkey=CDS]
MYGYGPGDPNLYAYQLQQMWTHIQRQEARIAKLEETTAAMHGEIEELKRQRGIHIDRIEYKFDQLKVEQLDGTLTIGVSP
STVHEIEDFAVNGMDASFPDSGTPPSGGFSPNENREPEQAGEGRPGADEGAANRHLHDFLDREAMEELIELAAKRHLDIG
PDESRRVMDDLRQQIESRVRQYASSPDLLHTGGQEPPGNATWKLIEKTKADIRAGIRNYLAQYQEGKGDGHP
>lcl|NZ_CP026520.1_prot_WP_009676346.1_5507 [locus_tag=PC41400_RS27795] [protein=germination protein GerPD] [protein_id=WP_009676346.1] [location=6147858..6148040] [gbkey=CDS]
MKVDFYVENKCLQVGSVYIIGVSSSSTFLIGDTDSITASSMFDTPPESVIVGPLVPLPHT
>lcl|NZ_CP026520.1_prot_WP_042235995.1_5508 [locus_tag=PC41400_RS27800] [protein=spore germination protein GerPE] [protein_id=WP_042235995.1] [location=6148044..6148529] [gbkey=CDS]
MSRRLVWLTQAKVLSVSSAAVFQIGDNGGIHTRFRALALQREIPVFLGDEGSFEAYSIFARPHPEIQFPPEAAKTTRNLG
TIQVGPVKVLGVSASSVMQVGGSGPVSLQSRIKHIRQFRPIGPYTNPTGTIAYTGSGENLEAESGPPEPGSSPSNPYGLN
R
>lcl|NZ_CP026520.1_prot_WP_009676338.1_5509 [locus_tag=PC41400_RS27805] [protein=response regulator transcription factor] [protein_id=WP_009676338.1] [location=complement(6148511..6149209)] [gbkey=CDS]
MKEGGRILIVEDDVSIAELERDYLELDGFTVDMEHHGETGLQRALVERYDLILLDVMLPGTDGFAICERIRVERDIPILM
VSAKKEDLDKIRGFSLGADDYIIKPFSPGELVARVKAHLSRYERLKGQQDQDKDEIRIRGLVINRASRRVYVNGKEIPFT
TKEFDLLTHLAMNPNRVFSKDQLFDHLWGMDAIGDISTVTVHIRKLREKIEPNPSDPAYIETIWGAGYRFKP
>lcl|NZ_CP026520.1_prot_WP_042235993.1_5510 [locus_tag=PC41400_RS27810] [protein=HAMP domain-containing sensor histidine kinase] [protein_id=WP_042235993.1] [location=complement(6149206..6150684)] [gbkey=CDS]
MSIRTRLLLSYIAMLVIPLILVGLTVTGLYFMLIQDLKREYHIDLTHGNPLKEVAQMEDDNYNKIRSSIAMEPGNAGDPG
QWKQWEALVERVNMGLIIRKDEQILYISPTMNHPDVLNSLPAFSYYSGITDSQLENNYYLLRHHDFTFPDGGKGSIFIGT
HDTIGKKYFLKYGLIFLAVLIAVLVLTNGMMTYLMSRSLITPIRALQRSAEHIKEGRLDIEVRPASSDEIGELAVSFEEM
RRRLKQSIELQMQYEENRKQLISNISHDLKTPVTAIKGYVEGIMDGVTNSQAKFDRYLQTVYSKTVDIDRMIDELFLYSK
LDLRKLPFQFEEADLTAYLKDCVEDMQFDMEKKGIRLTLELSSEEPLRVVIDRDKLRRVLTNITENAQKYRDKPEGHIRI
FVLERPEFVVIGVHDNGQGIPSDVLPHIFDRFYRGDPSRSSRTGGSGLGLAIAKQIVEEHGGTIWANSRFGEGTTIQFTV
CKNRNEKGRMAT
>lcl|NZ_CP026520.1_prot_WP_042235991.1_5511 [locus_tag=PC41400_RS27815] [protein=VOC family protein] [protein_id=WP_042235991.1] [location=complement(6150908..6151648)] [gbkey=CDS]
MTFTKLVDVFVPVTDLKRSAPWYEQVLGLRLLGRDDRHARLSFLEGTCLTLVQQTELNRYDEAPFNVNVYDAEKAYRSMI
GSGVEIAEELAPFHHMQCFKISDPDGNPIGIVGWREEAYAGTEEAIRLGGAFLPVRDLQRSVEWYESKLGALVLHRFAYK
LPALDKEVEAVSFANINVCLFETPRFSALAHMAFTIGSTDPRNDFETLRSGGIKLSALTREFNRDRFGLLDPDENEIGVI
GILQVN
>lcl|NZ_CP026520.1_prot_WP_042235989.1_5512 [locus_tag=PC41400_RS27820] [protein=ATP-binding cassette domain-containing protein] [protein_id=WP_042235989.1] [location=complement(6151768..6152745)] [gbkey=CDS]
MITATNLNKTFRTPVVRQGRMAGLRSLLSREYRSKTAVNDVNFTIGRGDFVGFIGPNGAGKSTTIKMLTGILHPTSGEVR
ISGLSPQRDRRRVVAKLGVVFGQRSQLWWDLPVRDSYEILAAMYRVREEDLKRKLAELGELLGLGEFMDTPVRKLSLGQR
MRADLAAAMLHDPDILFLDEPTIGLDVAAKQSIRGFLHKLNKEADKTILLTTHDMDDIEQLCERVIVINGGTIGYDGTVE
GMRQRIGLPTIMTIRYRGACGFAGGEADKEYPFRVTAAEKEKLVLEFNRTDIGPMEVLQLAGGWGEIADVHMQEPDFEDI
VHRVY
>lcl|NZ_CP026520.1_prot_WP_042235987.1_5513 [locus_tag=PC41400_RS27825] [protein=D-alanyl-D-alanine carboxypeptidase family protein] [protein_id=WP_042235987.1] [location=complement(6152778..6153698)] [gbkey=CDS]
MKRIKRKVLLLAVLAAAGVWLWTESKHTEPDIKATSALLMEAGSGRVIYERNAEESLPPASMSKMMTELIVYEEVKAGRA
SWEDQVTTSPYASEVTGAQIGLETGEKLALKDMLAAMAIVSANDAAVAVAEHLGGTEDEFVVKMNAKAKEIGLSGKTVFA
NASGLTRADLGPFAPKKNTGETLMTAKDAAKLAAYLIREHPGILELTAKADYTLPGRFGTLKTTNRMLPYTKDSFSYEGA
DGLKTGFTDQAGYCFTGTAVRDGKRLVSVVMGTETAEKRFEETRKLFNYGYGTRRSLTEWIKDMTR
>lcl|NZ_CP026520.1_prot_WP_042235984.1_5514 [locus_tag=PC41400_RS27830] [protein=ABC-2 family transporter protein] [protein_id=WP_042235984.1] [location=complement(6153918..6154712)] [gbkey=CDS]
MRNGLRMYVLLIRASLRSRMQYKFNFIFSSLMAALVNVAEFLMVALVLDRFGSVKGWSLYEVAYLYAVILLSKTIYRSLA
SDVHHLEKYLVSGDMDALLIRPLPVLFVLMTQNFRLLVGEFGQGLLVLVYAMTHMLRAGQIGWEAIPLTAVAIYSGALIL
FAIGLATASAGFWLTRIEELQNVTEDAARSAAQYPLDVYPSWLRYILLTFLPVGLANYVPASVIIRHEYGGWALAAVTVM
AFVFVLLAGRLWKLGISRYQSTGT
>lcl|NZ_CP026520.1_prot_WP_042235981.1_5515 [locus_tag=PC41400_RS27835] [protein=ABC-2 family transporter protein] [protein_id=WP_042235981.1] [location=complement(6154709..6155485)] [gbkey=CDS]
MLFSVLARKAYARNLQYRGAHMIHNVSSFMFGFVYVSLWAGIGESHSLGEYGVQGMTAYIAFNQASLWLVLFITNGLGME
ESVRTGQIALDLMRPVPLFLHKMYREWGQIAYQFVYKFLPIYILYFFLFSIPVPGKLSVYGWTALSLLLAAYISICMNYL
IGAAAMWTTESRWLYWVNYAFSMLLSGFFIPLEWLPAWLRQLSWLTPYPYLLYVPVSLFTERTTPAALPAALFWCVLFTG
LCMLATGAMRKKLEVQGG
>lcl|NZ_CP026520.1_prot_WP_042235977.1_5516 [locus_tag=PC41400_RS27840] [protein=hypothetical protein] [protein_id=WP_042235977.1] [location=complement(6155574..6155792)] [gbkey=CDS]
MNRNRTARAGLSIGALPIGLPPPKAEYPVHFIEENRLGGKILHAALQAFFFIGRPHGGGNGDNDGKLNLFRG
>lcl|NZ_CP026520.1_prot_WP_042235974.1_5517 [locus_tag=PC41400_RS27845] [protein=general stress protein] [protein_id=WP_042235974.1] [location=6155849..6156448] [gbkey=CDS]
MTRKARGVFQSSREAHRAIEALKEHGYRDEDISVLAKHREEFGPYTERQEIAGKEDTKAEKGAIGGAAIGSAAGLIAGIG
LMAVPGIGPILAAGPLAAALSGAAIGAGAGGVTGALVGAGIPEEEIENYHSRVDRGDILIVVDTPENKWELVNEILRENR
DDNFTADERSKSDVYEATPHAVHDDPDLLSNQEDVNFRR
>lcl|NZ_CP026520.1_prot_WP_129112547.1_5518 [locus_tag=PC41400_RS27850] [protein=YwmB family TATA-box binding protein] [protein_id=WP_129112547.1] [location=6156699..6157616] [gbkey=CDS]
MMRHAMMKSTRQQSPLTQVHRGRTPDSAAAGHPAATQQRRGVPARGLLWGIALLAALLLAGWSAAAKPTAADAGRSDLAA
LLAAADGVLAPDSAVTLRYAAPLPAPAGDDALAAAGDALAARLSLPAGAVSRASGHAVYAAGGGSAAGGGSAAGGGSAAG
GGGLSLQLARLDDGSAFVIARLEARAADGLPALEARRASLEAALQAAGVQADCNVVLQGAAARPSAGAAAQADGLQALLA
KRLGAAPLERYADGETVSITMSAPKLQASVISGGKRVNLQAAVHTVTESGLQRITIGTPVITTEY
>lcl|NZ_CP026520.1_prot_WP_042234999.1_5519 [locus_tag=PC41400_RS27855] [protein=hypothetical protein] [protein_id=WP_042234999.1] [location=6157976..6158533] [gbkey=CDS]
MTLRFFQWFGKWSLPEGFGRRLTAGLILTAVLVSLVLSALLNRLVPGSMTYADLMTRSQPASFYLPDENFTYRDQGVIES
FTRLALTKQALRKEEAASAFIPGKEPLRLSYDNSKGRVILVDADGRIYTPVSKAELRNESKLHWLWWKLDTGRSSYLYYV
TEPDAAIARLAEQIKNSAPAIEAAG
>lcl|NZ_CP026520.1_prot_WP_042235001.1_5520 [locus_tag=PC41400_RS27860] [protein=hypothetical protein] [protein_id=WP_042235001.1] [location=complement(6158768..6159811)] [gbkey=CDS]
MNAVQLPLSNPPLKGFLRWGYTLSITSGHAVTIPWFYSNFIQLSCTKKFLEDGRQCYVDFFRGQPNELNFNNPFLLTCNV
NYTFLENLALDDWPKFFADQIRNGYYCIVFLDESKLSPAACYQQEPFPHHLFLYGFNLEERVFYASMFDHTGVYRNLEIS
FEEFLDAVNSMKKLLKEGQTSDHHTYFYKYEPEFHYPFDKVAVIDQLRDYLNGETRLNRINYNPDENEAFGIRVYDYLQM
YYDAVEQNDPRLGMRNDVRHLHLLWEHKKMMSDRIEYLVTEGIIPYDEELAEGYKDLAKRAMKLRDQYIRHEIREDKEVF
ERLRLRFNQFRDEEPVLLQKLIALLEG
>lcl|NZ_CP026520.1_prot_WP_042235003.1_5521 [locus_tag=PC41400_RS27865] [protein=glycoside hydrolase family 3 protein] [protein_id=WP_042235003.1] [location=complement(6159771..6161429)] [gbkey=CDS]
MRACSELTLKEKIGQLLMVGFEGLTPDMNIRDLLENHFAGGVVFFSRNLDNPGQAFALTEELQRMAVAATGIPLWMGTDQ
EGGMVVRVRQGIAQLPAAMALGAARNPGLLYEAAKGTAEELKALGINMNFAPVVDINVNPRNPIIDVRSFGDSAELVSEL
GIAAMLGFQDGGIASVIKHFPGHGDTETDSHAELPVVRHSLERLRSVELVPFRRAALTGAEAVMTAHVGIPLLSGGEPVP
ATLSREVLTGLLREELGFEGLIVTDCMEMDAVTQGIGLGEAVVRAMLAGADLLLVSHTYASQLEAVAALERAVADGRLPE
ERIDASVERILRLKERRIGELRERSWEEARLLLENPQTLRTIERLREESITAVNREPDFCRLSPDVDTLVLWPQISAACK
SEDEPIHDATLAGFLRPFIGANLTERVYGTSPSPDEIAALASEAQRYGQIVAGIFHTASNPGQPALVRKLLEGGAKVIPV
SLRNPVDLAAFPEAEGFLACYEHHPHTLQALARVLTGKAEPAGTLPVTIPDTTTEGGEPDERRSASVIESAT
>lcl|NZ_CP026520.1_prot_WP_042235005.1_5522 [locus_tag=PC41400_RS27870] [protein=zinc-binding alcohol dehydrogenase] [protein_id=WP_042235005.1] [location=complement(6161449..6162474)] [gbkey=CDS]
MPDVAISGGYTSLNCVFQEPDRIELLEEKVPEPGPGQILCAAKKSLISTGTELACLKGRFDEGTVWSSWVRYPFYPGYSM
AAEVLKTGSGVEGIRAGDRIICPAGHRQYFLADYREAALIPDTIRDEEAVFTQIAKVALLGILRAEPAFGERVGVVGLGL
IGQLVVQYLNSSGARQIIAIAPESPRLDQARRNGATDLLDLPVADAYGQVEELTGGRMLDTVYDITGSYAVLPYCTQLTR
DLGKVILLGDSTEPSKQTIGPKVVFNSVSILGIHGRMMEGFKGWTEPDMNGFIFESIRRGKLDVESLISEKVSPLEAVSV
YSRIAGRKSDFMGVMFDWTQM
>lcl|NZ_CP026520.1_prot_WP_042235007.1_5523 [locus_tag=PC41400_RS27875] [protein=ABC-2 family transporter protein] [protein_id=WP_042235007.1] [location=complement(6162527..6163330)] [gbkey=CDS]
MFKIMRKYALLYGMFIKNCLIAQMEFRGNFIMSLLVESVYLLAKLLYVLVVFRTDLHVDGIPPEGLLLFIGMHTVATGIY
VGLFFTNFMKIPEYIKDGSLDLMLTKPVSLQFMASLRYVDLALPIPDILVGFVMVGIGWHAMDIPLTFVQLAGFALLLVV
SVVITYCLMIIPALLSFWFVQTGSVSEIAHSVWDANNFPMAIYPAWVRRIGTFVIPLFLVTNFGPMFLLGQLSWLYAGLA
LAGSLLLFAVVRLLWKQAVRGYSSASS
>lcl|NZ_CP026520.1_prot_WP_042235009.1_5524 [locus_tag=PC41400_RS27880] [protein=ABC-2 family transporter protein] [protein_id=WP_042235009.1] [location=complement(6163343..6164152)] [gbkey=CDS]
MNRRKLYMAVFSMGVQHSMEYRFHFFLGLLGAAFPILVQYFIWTAVYRHSGQAALFSYSYGQIILYTILAGLVSKLIATQ
FEHQIADDIKNGGLNKYLIKPVSYFGYRLVSFFGQKAIYYGITAILLIGIIWISAARDVLDVEAVRLILFAFTLLGALLL
NFLIAYCICASAFYLHEISYFFVITSLLVNILSGGMFPLEIFGETVVKALQYTPFPYTIYFPVNVLSGKTEAVAMYQGML
IQCGWILLFFWLSRLTWRVSMKKYSAVGG
>lcl|NZ_CP026520.1_prot_WP_042235010.1_5525 [locus_tag=PC41400_RS27885] [protein=ABC transporter ATP-binding protein] [protein_id=WP_042235010.1] [location=complement(6164149..6165132)] [gbkey=CDS]
MDFIRVRELHKSFVYYRKEAGLKNSLKNLFARKSLVKEAVKSVSFDIGPGECVGFLGPNGAGKTTTLKMLSGILYPTGGE
ADVLGYVPWERKNEFKRLFSIVMGQKNQLWWDLPASESIYLNKCIYDVDDDLYRRSLAELSEMLDVQELLNVQVRRLSLG
ERMKMELIAALIHRPRLLYLDEPTIGLDFPSQKKVREFLKYYNEQFGATILLTSHYMKDVEDLCKRTIIINEGSLLYDGD
LHKINDLFGEQKIIRLQFSEPVAEHRLAKFGKIVSGDGYNAVLELPKHRLKEVSRAVLDSFPIVDWTIEDVPIEESISML
YRKESAG
>lcl|NZ_CP026520.1_prot_WP_042235012.1_5526 [locus_tag=PC41400_RS27890] [protein=radical SAM protein] [protein_id=WP_042235012.1] [location=complement(6165193..6166368)] [gbkey=CDS]
MQPKTAVTFDQLTFRNMKRTLVPLAETGRKRREDPAYAAPVPYEIGIKLNNGCNLRCKHCFEWNPDGFHHGFAKEVKNDE
IEIPVVEKLLAETRERKSRVFLWGGEPLFYSKFDELAELLAKEDRYTTICTNAILIEQKLDSILKMSENLVMLVSLEGFE
KENDAIRGKGTFKKVIHAVQLLLDLQKQGVYKGKVSVALTVNDQMVNQLYSFMEYFEEMGVDSVYFCFPWYIPNDTAEKM
DTYFDAHFPHLAARFEEGHKNSWHSFTFHISPDNYDTLIDELNRVNSRVWNVRVRYQPALEPEEVEDFIRGGEKPAMKRT
KCFSISNRMDVMPDGKVNPCKFFPEFSVGNLNEESVADIFHGEDLRKQREVLACGLMPICSKCVLLYNNGI
>lcl|NZ_CP026520.1_prot_WP_042235014.1_5527 [locus_tag=PC41400_RS27895] [protein=hypothetical protein] [protein_id=WP_042235014.1] [location=complement(6166340..6167446)] [gbkey=CDS]
MNSSSVTLLASGNSLGAYIPAMHLHTYLQGRGVGSDVHVLENVYHEEVRDKIRATKKAFHADFSVARMGHKLARPVDTSL
NEEAVLRLLEDWMRAGVTRFAVFTGFWLPILERYKAAAGKPLDIRLIRLDAWDTPSFKVHKQLYPGYDNVWFYVPSELSP
GAYIASDEAEPLPYGQRAGRILIHGGGWGIGTYAETIRELRGLGRRLDVIVYDPSEVEEDDGVTRYFGTDASWDPWSRDA
QGRHTFPPMVRYVREQGVLREYPLRDYSAYTDLMRDCSAIISKPGGATLNDSLSFGIPFVMLEPFGDHELHNSAYWESCG
FGIRFAEWQARNFSEHLLEGICTRLLEQRSKIKHFGRITYAAENSSYV
>lcl|NZ_CP026520.1_prot_WP_042235016.1_5528 [locus_tag=PC41400_RS27900] [protein=hypothetical protein] [protein_id=WP_042235016.1] [location=complement(6167470..6168984)] [gbkey=CDS]
MSNSEARAIPKTVMGTFDPETRWRDPDLARLPAMPDKDRESIVLCMDDLLFPFCGPEDILYSRCKIDPVLHEYLEGLGFS
FRNRYLYDLNDERATLPGPDDFKRCMFRLAAGKDKLADPEKRVLQAAGSGYSDGAGLPQSFAPVNADGASGQAQSAEEAG
ITGAAGILSPYAITAETEAYLQAAGPLAPLPDLETVVRVNSKMYSNRLLARIGEKCYGTEASSAAEVEAAGNALLKQGPY
LLKDPFGVSGKGNLLIDSEAMQRRIAEHLHKQEGSGLRSHFLLEPLLKKETDFSSHWLIRESGEVEFISVQRMINHQQNY
GGSIKADESFIRLLENAGYFGVMEKALRLLAEDGYHGFVCFDSMVLEDGEVVSIVEINARKSMGLINAYLDKCWEKHGRT
GWLTFLSLGLPEGFAFGRLLHALRESGLLLTAPGGYGIVPLSSNTVMVNEILSRRRMAEGTLNKRGMPKGRLYVSVVGRD
DEHRSELIESLRHVLADLSVKVYN
>lcl|NZ_CP026520.1_prot_WP_042235018.1_5529 [locus_tag=PC41400_RS27905] [protein=acyl carrier protein] [protein_id=WP_042235018.1] [location=complement(6168971..6169246)] [gbkey=CDS]
MYDKIVDILCAIKEDQPELRHSLSPSTDLNNDIGLDSLQMIQFMLKVEEQLNVAIDYDQFDYEHLESIEAFISFLKSCQS
QAQPQYEDVQQ
>lcl|NZ_CP026520.1_prot_WP_042235021.1_5530 [locus_tag=PC41400_RS27910] [protein=hypothetical protein] [protein_id=WP_042235021.1] [location=6169973..6170206] [gbkey=CDS]
MIALLYSFLSIAALLTVRFMNFSPAVTLTTACLLIAAAFVCAYLDYKRNKGFNAYMKNVSITYLIFIVIFAYTAVSL
>lcl|NZ_CP026520.1_prot_WP_042235022.1_5531 [locus_tag=PC41400_RS27915] [protein=carbohydrate ABC transporter permease] [protein_id=WP_042235022.1] [location=complement(6170310..6171203)] [gbkey=CDS]
MPQMTGTAVPPVKGTGPARRENAFGKLAKSSISNILLAVYALVTLYPLIWLFISAFKTNDEFFGKPFGFPAEWKWNNFVQ
AWKVADIETAVINSAIVTLVALVLTLAIGALTAFVLSRFSFRLRGPLMGYFLLGMLIPVHSTLVPLFIMLKKIGMLDTYG
ALILPYVAFELPVAIFIIAAYMTSVPKEIEEAALIDGTGYWGIFTKMLLPLSVPAMSTVGILAFLRYWNDFAFALVFINK
PSLKTLPLSLSQFATGFGTDYSLTMAAMAISVIPTILIYLAFQEQIMKGMVAGAVKG
>lcl|NZ_CP026520.1_prot_WP_042235024.1_5532 [locus_tag=PC41400_RS27920] [protein=sugar ABC transporter permease] [protein_id=WP_042235024.1] [location=complement(6171223..6172101)] [gbkey=CDS]
MTSSSRRTNFIVLGLLPALLIYSVFVIIPIFWSAYYGFFDWKGIGEAKFNGFDNYVEALKDPIFWLALKNNLIVVAASVF
GQVPIALVLALVLRKSNFFQRFIRSAVFMPMVLSSVVVGIIWSYIYHPQIGILNFLLEAVGLESWKKAWLSDKSVSMLAL
TVPIIWNYIGPYLIMFIAALQNIPSDIDDAAMIDGATGSRKLFSVTLPMIWDTVKVTIVLCISGSLKAFDLIYVMTGGGP
AHSTELLASYMYNNTFSVYRFGYGSAISTLIIIISLVLIAGSQRLMRRNQAV
>lcl|NZ_CP026520.1_prot_WP_042235026.1_5533 [locus_tag=PC41400_RS27925] [protein=extracellular solute-binding protein] [protein_id=WP_042235026.1] [location=complement(6172469..6173794)] [gbkey=CDS]
MMKKTAKTGIALILSLSLLSACSKEPATTSTGDAGKGTEKVKLSLWHNFTGEDARAKTMRAIIEEYKKANPNVELEVQAI
PPDGYKTRLKTVAAANEMPDLFQMWPGAMTKEFVSGNLLQPVGDLLDKNAEWKNGFLPNSFDDFTVDGKVYSIPMSLSPT
SILYYNKKMFDDNGVTVPKTWDEMMKAVETFKSKKITPIALGNKAAWLAQSSILSSLADRYTGTEWFMKAAAQDGAKFTD
PEFVAALKALQDLGKAGAFQEGFNSIDNTQMEQMFAQGKAAMMIDGGWALTNLTNGAQPEALNNIEVTVLPSVPNGKGDP
QSLSGVVGIGLGLNKKVEGAKKEAADKLIQAMAGPDAQKRTLESNQLVAYKVELDKSKVSTLFAKVNDLVGQVKRTPVYD
SQLSSAASEVINNGIQELLMGGKPEDIAKKLQDAQAKALGK
>lcl|NZ_CP026520.1_prot_WP_042235029.1_5534 [locus_tag=PC41400_RS27930] [protein=response regulator] [protein_id=WP_042235029.1] [location=complement(6174139..6175749)] [gbkey=CDS]
MNVLIVDDEVIIRTGLSTVIQWEELGFRLLPAAESAEEALERAPAEHPHIVLTDIRMGGMDGITMAEKVKQMLPDSEIIV
LTGYDDFAYVQQAIRGGIGDYLLKSSRPEEIIRAVMKAKQRIVSKWDARKADNMQRRAFRSRLLERLATEGIDDADSLEQ
VPQVLAKLGLPHLPEASAGVCLQVLLISVQGWGDSRAAGSLLQFAVDNMLGELLPCETLRRRHDIVVLLRAEGRDAGALR
SKLGRISDKLKCVLFTAAGSRVTGLGQVPRSYREAVYTASFRGFAGTETLLAYEAVEGREGGRTVCSKEEESELAILLKN
GNPIELRHFVQKALDGEIADAEATPVSLKAYSDSLVVAAHRWLERTAAALGMPQLPRLPEQDPAKRAQGDKHPGDGPGAE
GQGDLQENLYLRLRVIMETYYQYVSEDGVLYIKRAIAYIQDNLDRTLTLQQVAGHVHVHPNHFSEVFKRETGVNYIEFVM
RERIRRAAEILSETPVKVSEVAKKVGYEDIKYFSQLFKKYTGQTPTEFRAGREGKQ
>lcl|NZ_CP026520.1_prot_WP_042235031.1_5535 [locus_tag=PC41400_RS27935] [protein=sensor histidine kinase] [protein_id=WP_042235031.1] [location=complement(6175746..6177506)] [gbkey=CDS]
MRRWLDRSLKRKLSFLMLVSIGIPLLSFGIFSYQIASSVTEEKAKQSGFGLLTQMRTNMELMVQDVENMSIFLIGQRDVQ
QYLSDTQWDALRPTIIIGFLTNLAFSKPYIAEITIESARDLPSLSNTSIVSSGMPGWPGPKDTEPGNRPKWWTPLYKNQT
TIGEKDVISLVRPIRSVNTFAHLGSLTISLDQKVIADTLRKSNISEGGSVLLVNEQGNIIAGPGVEWLNRPLSEWMPGMS
GITGTSGSFNVGQGDDRNTVLYDTVPGVDWKLVGIMPYKNFREQNRYVLLLTAAAVAIAVVLVASLVIFFVQRVTKPLLM
LTRFLKDTDPEEPMKALPVTSADEVGQLMRSYNRLSDRIERLTEQVKLNEARKKETDMLALQAQIHPHFLYNTLSSVHWM
ALMSQDMKIAEMVGHLSDFLRFSLNKGAEYCEVQQEVAHAEHYAAIQAVRYPDKFDIEFVVDPDIRQKPVLKLLLQPLIE
NALIHGIQKQREKGSIRVYAQRMGTELTFMVEDTGIGIEPDKLNEIRAALSAADEPVVPGSYGLRNVHRRLQLHYGSEAG
LHIRSIVGEGTVITFMIPIVERAGTA
>lcl|NZ_CP026520.1_prot_WP_042235033.1_5536 [locus_tag=PC41400_RS27940] [protein=cobalamin-dependent protein] [protein_id=WP_042235033.1] [location=6177725..6178807] [gbkey=CDS]
MNHEETGRRLFQMAENLAEYVTQLQYKYQPELTERFGERGRLLTIQDSGYSIKYLAESVYMQSPDLFNHYTSWMTELLSG
YQVSREDLALNFRMIEQAINAHFGEADRALIGEYLEIGLHQIQSAQTSETFLSEERPFAETAIAYTEAILRADRREALKL
ILTKLEEGMAIADLYLHVFQPSQREIGRLWQTHQITVAQEHLCTAATQMIMSHLYPHLFATERSGNTMVAACVGQELHEI
GLRMVADLFELNGWDTYYLGANVPTRSLIRTLVEKKAQVVAISATMTFHVHLVEELIAEIRSSEAGQYVHILVGGLPFNI
DRDLWRKVGADGYGDDAQEAIRLADHWKHHRIGNELARSN
>lcl|NZ_CP026520.1_prot_WP_042235035.1_5537 [locus_tag=PC41400_RS27945] [protein=ATP-binding protein] [protein_id=WP_042235035.1] [location=6178841..6180583] [gbkey=CDS]
MDPNMKTEDERLEITHYRQTIEKLSQQLIQARQREEQALTQFSSLNNELITLQRQLSKTNVELVRSKAEAERANEAKSMF
LATMSHEIRTPMNGILGMAELLADTLQTEEQKQSIAIIEESASLLLTIINDILDLSKIEAGHMQPEPETFEVRKLLNHSL
RLLEPNASRKRNKLRGTVDESVGQFLTGDAGRIRQILINLAGNAIKFTDGGEVHIRIRLLQSSATFQTLRFEVTDTGVGI
PEEHVNRVFEPFYQAHTQREDAQKSTGLGLSICKRLVEWMEGRIDVTSKVGTGSSFWFELQLAIPSEAEYRQEAAAALQT
TGSGIGPAFTDEERSQPLLLVEDNAINRQVALLQLQKLGFMHIDTASNGEEALEARSRKDYSLVLMDSQMPVMDGLQATQ
LIREWENARRHPRSAIVALTASAMQSDRLRCLEAGMDDFLTKPIQLSKLQSVIRKWLPAGPPEPVSEHVLIPEVIQELNG
LDEPGSPGVLAMLLGMYQEQTPSKLNQLEEKMRAGDLTAVQYVAHDLKSTSLSLGIDQFSKLIERIENQAKNGTVHLSGT
IFKELWQAYDKACRELKALL
>lcl|NZ_CP026520.1_prot_WP_042235037.1_5538 [locus_tag=PC41400_RS27950] [protein=nucleotidyltransferase domain-containing protein] [protein_id=WP_042235037.1] [location=6180809..6181426] [gbkey=CDS]
MNNEHIKKAADYLSDKYEGHSVILYGSYANGDYTDESDLDLMIFSDLNFKRNDTSRIDGLLLDAWIHPTSSMDDPARFLH
IHDGVIVTDSRGLAAPFLEKIQSIFAKGPEKSGPEEADFQKAWLQKMLKRAGKGDAEGDFRYHWMLTDSLEIYFKLKGRW
YWGPKKSLRWLEQHEPGTYLLFRKALSPLAGQEDVSELIGHIVES
>lcl|NZ_CP026520.1_prot_WP_174446855.1_5539 [locus_tag=PC41400_RS27955] [protein=GTP pyrophosphokinase family protein] [protein_id=WP_174446855.1] [location=6181670..6182290] [gbkey=CDS]
MKSWAHFLLTYKFALDEVGTKLQILNEEFQFIHDYNPIEHMKTRIKSAESIAAKLKRKNLEITMDNTRSYIRDIAGIRII
CSFSTDIYRIYDMICSQNDVQVVQTKDYIKHPKPNGYQSLHLIIQIPVFLTNRIEHVPVEIQIRTVAMDFWASLEHKIYY
KYNEGVPAEIRQELKDTADLIASLDARMLALNNEVQSQKALVKPEA
>lcl|NZ_CP026520.1_prot_WP_042235042.1_5540 [locus_tag=PC41400_RS27960] [protein=potassium channel family protein] [protein_id=WP_042235042.1] [location=6182454..6182777] [gbkey=CDS]
MVSFILTLKRLVGGLWHAFKEKDFQVLFVLILLILLSGTIFYTQVEGLSVLDSLYFCVATLSTVGHPDFVPQTGLGKIFT
IIYIIAGTGLFLGLILRVAYGIMTSKK
>lcl|NZ_CP026520.1_prot_WP_042235043.1_5541 [gene=thrC] [locus_tag=PC41400_RS27965] [protein=threonine synthase] [protein_id=WP_042235043.1] [location=complement(6182906..6184159)] [gbkey=CDS]
MKFVCVECSKVFAEFRYTCTCNGMLDIFQDFSGLDAEELKRLFQSRLSDRLSPYASGVWRYKELIFPDVEDSAIVTKYEG
NTGLYRSDAVASYAGTRSVWLKAQSENPSGSFKDNGMTVAVTHGRSLGYDRFTCTSTGNTSSSLAMYASLAGRKSYVFVP
NKNISLNKVLQTLAYGANIFSIEGTYDHGIKFLEENSEKLGLYICNSINPFRIEGQKSIVYELAQVMSWKLPDWIFVPGG
ALSNVSALGKGLSDLYALGLIDKLPRVALVQAEGASPFYKMVAEGKTALEAEPNPYTRASALNIGDPPSWRKAQRTLHET
NGVAVSVTDEEILNAKAVIDASGIGCEPASAATVAGLRKLVSQQIVDKDETVACILTGNMLKDTDALREYHLEDNKPFSN
KIIPSSLTYDDIFAVIK
>lcl|NZ_CP026520.1_prot_WP_042235046.1_5542 [locus_tag=PC41400_RS27970] [protein=histidine phosphatase family protein] [protein_id=WP_042235046.1] [location=complement(6184344..6184979)] [gbkey=CDS]
MAVTTLYLTRHGQTEWNVARKMQGHQDSPLTSLGKTQAAWLGEALAEVPLDRIYSSSSPRAVRTAEIIRGGRKLDIAERD
SLREMNLGEWEGQSLTDIEQREPERHHAFWKAPHQFRAGTGESYEEVKERAVGEVLRITEENPGKSMLIVTHTVALKLIM
AHFESRDLRDLWELPYIHPTCLCKVELTDGVPRIVLHADISHYKNEIEGAF
>lcl|NZ_CP026520.1_prot_WP_042235048.1_5543 [locus_tag=PC41400_RS27975] [protein=GNAT family N-acetyltransferase] [protein_id=WP_042235048.1] [location=complement(6184970..6185458)] [gbkey=CDS]
MENLTLQQVDAKHPDLWLLIEKLDEELLQRYPAEDIFGLDRDDPRLDQSVFVVAYGGDRPVGCGGIRPLDAESTELKRFF
VDNEYRKRGVADRMLAFLEERARERNFVSVKLETGPEQPESLYFYKKHGYVEIEKYGEYIGSEASICFEKPLAAKGEGIT
WR
>lcl|NZ_CP026520.1_prot_WP_042235049.1_5544 [locus_tag=PC41400_RS27980] [protein=multidrug resistance efflux transporter family protein] [protein_id=WP_042235049.1] [location=complement(6185780..6186745)] [gbkey=CDS]
MKPILLGILSAFFFAITFVLNRKMELGGGSWLWSGSLRYLFMVPFLLLIVAGRRNLRPLFRAMRLHPWKWLLWSTVGFGL
FYAPMCFAAAYSPGWLTAGTWQLTIISGSLLAPLFWETIATPAGPVKVRSRIPLRGLAMSLVILAGIFLLQLEQAGSLTA
KDLLLGVLPIVVASFAYPLGNRKMMELCGGGLDVFQRVLGMTLASLPFWLLLAGWAYASAGLPSTEQTFQSLIVALSSGV
IATILFFRATDMVREDVSKLAAVEATQSGEVLFALAGEVLLLSADLPSALSWLGIGIIVAGMVLHSHVSHRRPKAQFSME
K
>lcl|NZ_CP026520.1_prot_WP_042235051.1_5545 [locus_tag=PC41400_RS27985] [protein=GntR family transcriptional regulator] [protein_id=WP_042235051.1] [location=complement(6186752..6187426)] [gbkey=CDS]
MKTKPTIQRIKRQSFRDEIYEALRQAIVSLELEPGAKLSDQELAEQFGVSRTPVREALKRLEDEGLVEAVPGSATRVTPL
KPEEGRHAFTVVAALHALAARLAVPCLGPEEHEALEQSNKALEEALGRNDVPAAIEADNAFHAVFLQAAGNPEILSALKG
ILPKVQRLEISRFASVHGLQSVEQHRRIADACRKRDAAAAARLVEENWLTLGERLAGSNQAEKG
>lcl|NZ_CP026520.1_prot_WP_241688750.1_5546 [locus_tag=PC41400_RS27990] [protein=ABC transporter ATP-binding protein] [protein_id=WP_241688750.1] [location=complement(6187497..6188216)] [gbkey=CDS]
MAETWNGTEPIVAFDGVSKSFIGKRVLKDVSFRIPRGQIIGIIGTNGSGKSTVLKLMAGLLQPSKGRVELEGQRVSRISS
RRIAFLPEQDVFYPTHTVERTLRFYTAMYQDFDIAKALELTESFGLDLKEKVARLSKGNRARLKIVLVLARQVPLIVMDE
PLSGLDPLVRESIIRTVISYVDMEEQTLVLSTHEVDEIEPLLDRVILIRDGEVVRDEAVEQIQSEYGIGLVNWMKQAGG
>lcl|NZ_CP026520.1_prot_WP_009671720.1_5547 [locus_tag=PC41400_RS27995] [protein=GntR family transcriptional regulator] [protein_id=WP_009671720.1] [location=complement(6188244..6188609)] [gbkey=CDS]
MPEQFSGSQPIYLQLVQRVCRQIIRKEFPPGGKLPSVRELAVQFGVNPNTVQRVNMELERMGIVESRRGQGMFVTEDESL
LTRLRDELKEGELKRFAEEMRDLGYSNEQIIENVQKFLKLN
>lcl|NZ_CP026520.1_prot_WP_042235055.1_5548 [locus_tag=PC41400_RS28000] [protein=hypothetical protein] [protein_id=WP_042235055.1] [location=complement(6188614..6189387)] [gbkey=CDS]
MNAWWNLFKKEIGIRTAVTGSQKTWAAFSVGMIALMTFMAYRYQSGIVTPLWLGIIYLHVFLPSVYLLISFGKERERAPL
WFQLPQPGWKLIGAKYAAALTEMIGGLLVSITMFLWMYKVEKTALIERAGDDLFYLSGHVSYFRGLLHGSGLKEVFIFFF
IAFFIAALAVLVYFTALALSNRVGRLRWLAAIILIGAAMLVEYTFEMSPVYQFLFKWGYAFDWDGDPFYSGEMIWSLANL
AVIVYIPAWLLDRKVEV
>lcl|NZ_CP026520.1_prot_WP_241688751.1_5549 [locus_tag=PC41400_RS30500] [protein=hypothetical protein] [protein_id=WP_241688751.1] [location=complement(6190254..6190406)] [gbkey=CDS]
MKTRTPEIWLIKSDKAANTYMDEVYKESAFTGCHKGYTMVPQKNPSLLWT
>lcl|NZ_CP026520.1_prot_WP_042235057.1_5550 [locus_tag=PC41400_RS28010] [protein=hypothetical protein] [protein_id=WP_042235057.1] [location=complement(6190922..6191302)] [gbkey=CDS]
MHSIHDNEIISYKVDFQKSRITLHTLGGNGDSVTTTIEFIDVLAHMFETQLQGSIILDISKYEVSRFLKHNKELLEKQKN
YGWPMHYEKIEDLLERLQNEKYNYYVISSSYGLNGWVIAKEYNIYN
>lcl|NZ_CP026520.1_prot_WP_053228795.1_5551 [locus_tag=PC41400_RS28015] [protein=RHS repeat-associated core domain-containing protein] [protein_id=WP_053228795.1] [location=complement(6191338..6196467)] [gbkey=CDS]
MKKKINYLVLFCLVVQMLAPLYVIHANEQSASNPQQISQLLTVESLSKTYGVPETDLLSELNKGYSLSEIQSALQTRQDP
SQPLGEILNQMNPTIEEQLKKEDYSAERFAEKHPQPGPTAPTVSEQTYGTYVEPTGDSIIPDPSLLYPPGVTDATYGRAS
LLSSNKYPTSYDELAVKRLDIKADRAPSASSISEDVSTVNGSLQIQTTDMTLPGRNGLSFELKRKYDSSDALYYEKDILN
GSIYTTRYYPELTSQLYFKYSNQPVTENGSASDFYFGPAYQSFVQYMGEYTYWYFPANYPVIEEFNNEVYDSFKKAWPYM
DPKNPNTPPFIAKENIRLFGLDFYAKAYTTGNVRQDPMMVPEFRGTAYANKTKPIKNENRYPIGKGWSWDLPYIESKDNG
KTYIRLFGGATYELDGRTLKGYPWKDLTMYYDSSIKVNNLPSSYRLESLDGKKQYFAYNGKLIQITDAYQNTIQFEYQDV
TPYGQVLTKIKDAIGNEISIAYTEKDVTLTSGDRTVKYDKIKDPQGNKELLSQVTDPAGRKTQYVYSIEAAPFDLVRASS
MKDNYVALLKQVYHPTKAKTDYTYETFTRSLGVYAKETVHRVKTREDVVTFADGTESRSNRVDYAYSGDGGAVQERNTSF
ATTINNGRTQTTHNYDKVYVDDKTPEIFYNTQIKQDDGTQQTIQTMEYNRANRWPSPIKTTTKTVQGSANSTEQVVQRTY
DEYGNVLTETDPANAVTTYQYDASTHLVSSVTKPVQSGLNSYTELERYPTTNGIKVVRIKENNAAGAVKAQTSYTYDAYG
NPVQISVKDDTKDTIVNQIFGAQYQGGFLTEQSVEVTDAAKQNSVITQRFEYNPSTGSMTKFTDGKGFDTRYEYDKLGRT
TKVTQPDTSYSTYTYNDEANEITTVDPTGVTAFAKWNPLGWKIKTGIVGKPAQEFGYDSYGRQVWSQDGAGNRTSYEYDK
RDRLVATTYPGAEQAKSTIEYDDVNRIVTSKDGEGNKVKESFDLLGRLAQKEVYSSAGTLTESIRYSYDSAGNLATVNDG
TGNNPGDVQTTQYGYDVMNRLISVTDPETNTTRYTYSLANKLTQIEYPDHNKTQKAYDQMGRLIQETDPLGQTETSFYDA
GNNLIKAIDRKGNAQTFTYNSLNLQTTSVTGDESLTFQYDAAGRRLSMQDRTGTTNYLYEPSTGWLKQVQYPDKRTLQYA
YDGQGNRTQMTDPFGVVTNYQYDARNRLKAVGQDPMNWDTEYQYKNNNLPAVIQGKNGLVSAYNYEVSNLTSMTQNLSGT
GINSFGYGYDLHGNQKTKTENGKSNAYTYDLLNRIATASEFNENYSYDSRGNRLTLQSDKPITSADMSYSYDERNRLSEV
KVGEGTGVRYTYNGDGLLTERTENGETSRYYYDGDQIIAEATMTNGTVAHKASYIRGIDLATPAVYEDKEDEGNGQESSS
NTDIWGRTVIVQTEGKQSGVTHFSQLVAKVDVQGNKAYYVHNGHGDVIELRDAGGALLNQYTYDLWGNPTMVQEKVSNPF
GYSGEYWDNTAKLQYLRARWYDPGMGRFLNEDTYQGQLNSPLSLNLYTYVLNNPLRYIDPSGHMDMQAGSESVSAGGRIN
SGSAFQGGGGGGRIGRLASEKDLRSQGKPYENHTTVKKTDTTRSLPTTGKPNSSVDFYDNGKLTQRRYYDEKGRAKQDID
YEHSNGDNSHDFPHSHTWDWSSGSPKRSK
>lcl|NZ_CP026520.1_prot_WP_042235059.1_5552 [locus_tag=PC41400_RS28020] [protein=hypothetical protein] [protein_id=WP_042235059.1] [location=complement(6196577..6197857)] [gbkey=CDS]
MKKTISIFLILQMISVISMFGVHAQEIVSNVYDSKGRISYSSFISNNKFVEYKYDKNGNTIRKNLWGDNAYKKMSSIIGR
NESFETDNNQDGIADDWLISGQVKASMSSDSKYGKYSQRLETIDDGGAFRQELGNLSAGRHFLVMLDYKANASNTVSMRL
NDTQTDWGLNLGYNYFITDNTWRTAYVKFTSTGKPINLVAAYFDPGSHVVNIDGLRLYEITTAVYDQINSNPEYTGDSLI
EKFSYGDLLDYKGDFELDDNNDGNANGWNVDGHSVSSLSHESVYGNTSQQLELDGEWASLNQELGTLTEGSYYLFLVSYK
ATSSTTVSMRVNDTQTNWGLNLGYKYFKTDDTWRRAHIKFQATGNPITIIGAYLDSGVSKVQVDGARLYKISESIYNRID
NEEGYSGDLLYLKYPYNSFNEILGDT
>lcl|NZ_CP026520.1_prot_WP_042235061.1_5553 [locus_tag=PC41400_RS28025] [protein=NBR1-Ig-like domain-containing protein] [protein_id=WP_042235061.1] [location=complement(6198063..6198698)] [gbkey=CDS]
MIKKTIILCMIWTLAAINFTCTMQANGNTAESDASIVSYLIPSVMVVGKTYPVSVTVKNEGRTSWTSKDFFRLGAIDDMD
PFSEGRQYLPDGKIKKSGEEFTFLFSLQAPQTPGSYVTDWKMVQESVSWFGEALIKKVEVTTQQERGAVILSESIPTKMG
PGTSYSVSVTVRNEGTTTWTDNYRLGAVGDSDDFAKKSRLFHSVIFLVDVV
>lcl|NZ_CP026520.1_prot_WP_129112548.1_5554 [locus_tag=PC41400_RS28030] [protein=carbohydrate binding domain-containing protein] [protein_id=WP_129112548.1] [location=complement(6198709..6200304)] [gbkey=CDS]
MKIFASTLLVVFGLLVSNLSVFAVSYQYSDNGQLEYMQLDSDLAREYSYDKNGNLLQAKEHQTVYPGSLVRNGSFEDGKD
EGGWTSWPGDSTVLLDNDSLRGDKSIKIVNKNKSYSNWGQTITGFIPGQTYLMEGWVKTDNVTALPNHGAFLYVSFKNAE
NNELAYKWIDPITGTNPWKRLGAAITVPEGTASIYISAMLYGASGTAWFDEIKINEVNPNLIFNGGFEKESGRDDNHWGS
WSPETSNIKADNAGHTGANSLLLQNKDMAYTNWGQTIKDIKKGRYYLFEGWVKTENVDAPENYGAFLYYSIRDAAGKEVS
NEWIQPQTGTRDWTRLSRIIKIPREAKEIYISTMLYGASGKAWFDDMKLIEMNDKGDLIQNGSFEWEYGIKEGGWWSWST
TDNISQSDLARDGKKSVKLQNPGQAYTNWGQTITDFKVGEKYTLEGWVKVENVQAPPLYGAFLYYNFKDANGKEIANEWV
EPQVGSKDWTKISKDVIVPAGTKEIFVSTMLYGASGTAWFDSIKFYPASSR
>lcl|NZ_CP026520.1_prot_WP_042235065.1_5555 [locus_tag=PC41400_RS28035] [protein=hypothetical protein] [protein_id=WP_042235065.1] [location=6200484..6201080] [gbkey=CDS]
MNRKIGLNSLVSMICLSLLLAFAGKIQAVPTVMTGTKQLYGSGGQLDYTHLPSNEFIKYKYDRNGNLLGKEPANMEPDSS
VDLNGTDVYEIPNLNVDTSFGSKTTVEFWMFWDGKEVVMPFAWSSKFDLYFADGYFGFNHFNGDIFGIPSTTLKNKWVHV
AAVFYNGDPDSGNLELYINGVKQTLQKALGKETWGESG
>lcl|NZ_CP026520.1_prot_WP_042235066.1_5556 [locus_tag=PC41400_RS28040] [protein=sialidase domain-containing protein] [protein_id=WP_042235066.1] [location=6201062..6202390] [gbkey=CDS]
MGGIGVSNRAYIGGYKNENGTDYRFKGKIANLRIWNNKQENKQIVTNMYKNLKGTEDNLIGEWFYEKTLINPKSFNGTKF
IELSNLDINKQEGGYNTVEFWMFWDDTNTVMPFAWNTRYDLYFADGYFGFNTFNSDLLGIPSLPLKNRWVHVSAIFKNGD
VTKDSCELYIYGTKQNIQNHLGSDKSRASAGDKVYIGGYKENNQTLYNFKGKLANFRIRNKKLDVLQIQSNIFKRLNGNE
SGLVGEWGLRDNPLNPKIFKGDMYSELSNVHINTGQGEKNTVEFWMFWDGTNAVMPFAWDSGYDLYLADGYFGFNTFNAD
VLGIPSDNLKNKWVHVAAVFYNGIPDKDNVKLYINGLPQSLQTLKGPFTRGGRASTNVYLGGYLEAGKKNYLFKGGITNL
RIWKKELSEDQIKTYMYLNSAKAPDLVGQWKLENSEYLLKEK
>lcl|NZ_CP026520.1_prot_WP_042235069.1_5557 [locus_tag=PC41400_RS28045] [protein=S-layer homology domain-containing protein] [protein_id=WP_042235069.1] [location=complement(6202587..6203906)] [gbkey=CDS]
MKTSTGLRKKMAVTLLACTVMAPAQLSWAASASAASDISGHWAEQQTKQWVDKGYIQGYADGTLKPDKAVTRAEFVSLVN
RSFKLEAQGTASFKDVAASDWMYGEVAKAVQAKYIEGYPDGSFRPNASISRQEAAVIVARLLNKTAAQSETAEAFRDQSA
IASWGSSAVAAVVSSGVMSGYADHSFKPEAPITRAETIVTLQRAAGETTAPTETTGPGTEEGAQLAESTIAITGPDSVKL
TDESAQSISFEAAVKNKNGEAVKDAKVVWSLEFANDHQRWQDEADAPYGITIDPATGKLNVGWYSWIEDFKVVAKLASDK
SVVGSKTVHASQGNIKNVLASNGKVQFVFFEEPEVLDDGYGGKFRADIDSISIAQEDLQDPSQYKELDITKTDYDPATKT
ATFHFNPIAAAAEKQDIYLSIGYIESTYISGAHFVIPAK
>lcl|NZ_CP026520.1_prot_WP_042235071.1_5558 [locus_tag=PC41400_RS28050] [protein=glycosyltransferase] [protein_id=WP_042235071.1] [location=complement(6204124..6205320)] [gbkey=CDS]
MKITLLPYDDPEAGKPFAALASALQAGGYEAGVVVPAGSIPYPAHSGTGRALMPRRVWKLPGLKKDGLEKEGAGDGYGNL
PEVCAESDLIITDLRMLESALAVSDELNSKLIAVVLTPPCPVSRSAAVVSAALKKLPFFYGAVRKREAGYAWPLTCKAVQ
DKLILRDIPVLHAYSACFLKRPGEWGGAAGRIVTGEWITTMAGTGDFICSSKSCEDTEQWLNEGEAPLYFGFSGLPLQAQ
AHCIETALDAVREFSCRAVIEADWEALGRLRHELPGYALPIEEADYDRLFPRCRSIVHRGSAAAAHASLRAGIPALVCPA
AAEEKVWGRLTATTGAGSCLPLNKVAQKKLTQALKVLEHKRMKNKAAELGIGLRRENGIETAISFIDRYLSWTQYRKF
>lcl|NZ_CP026520.1_prot_WP_053228796.1_5559 [locus_tag=PC41400_RS28055] [protein=stalk domain-containing protein] [protein_id=WP_053228796.1] [location=complement(6205386..6206048)] [gbkey=CDS]
MKKIMAAAVLGLLMIMSILAYSLSFSLDFLSSQKFVVNGKQAEYSVIEYNGHVYVPLRFVAENMASDVKYHENTGITSID
QHSIRFDDAVAAPKPEVVYEKRLRNGSMWMQDIPLLQGNSCWVECIQDSRSDLTLLKQAEYPAVEVKPESYIGIRYTKGM
EPQQLECSVITSADGKELYTAIEMDQGRIRLPREPGVYSIVLRSNWEQGNTTYLFAVKIV
>lcl|NZ_CP026520.1_prot_WP_241688752.1_5560 [locus_tag=PC41400_RS28060] [protein=extracellular solute-binding protein] [protein_id=WP_241688752.1] [location=complement(6206107..6207456)] [gbkey=CDS]
MMRKQVGVVLICAALATSACGAGESGKTGGTSNKEKPKSKEGKTVVTLSLRNPDPLFQVIEKKFEEKYPDIDLQIQSYKQ
PGEQWGAGDYEKYVKTTNTALLSGKGPDIVELSDLPVDKYVHSKLLVNLNEMFDQDKSLKKDEMNLNILNGLKKKDGGLY
TIPTSFYLRTYIGDGDSIKKANVTIDDANWTWDQFGETARKVREKDGGGGKAKRPVFADDPGVILQEIIMDRYGELVDPA
TMKAKFDSNSFTDLLKSVKKMYDDKVMTSEIAEGETALFYSAAFLTLQDLVAQPYVLFSNPKLLQKPHAEGKGGIRIIPT
DEFGIQANSPVKAEAWTFLSFLYSKDVQSLQDLKGFSLLKEVNDKKLDEIQKQVKAGTFKLPTGKAANISDEQFNLYKKL
IQKADSYSAFDMKVLSVINEESPAYFSGQKPAEEVAKLIQNRVTTYLNE
>lcl|NZ_CP026520.1_prot_WP_042235073.1_5561 [locus_tag=PC41400_RS28065] [protein=biotin/lipoyl-binding protein] [protein_id=WP_042235073.1] [location=complement(6207524..6208684)] [gbkey=CDS]
MEQEASKDLKRKRNIRIIAGLFIGGLILFTLFSNTLQALTLPKVRTESAVYGSLVHTLEGTGVLRPLAEEKLSNSAGWKV
KAVLVKEGDRVKKGQKLVSYDSKSAERELQDETTQLEKLKIELQNAQDQFIAASTEGDELKVRSVTREMETRKLDISMQE
RKVTGLREALADQKDLAAPFEGVITKVNASAGTKPAGEPEIVIADSAKGYRFEFAADAALLTSLGITAGTKIEVGLGTPD
QQPRLIEGTVEETAEAEPLAAGSSGAEAGESGGRTAARKSLRIQVEGPELKGGDRAVVKLTKRSVQQGFKVSSDAVHQER
DGKFIYKIEEQKGALGNVFVVRRVPVRSSETNESETVIQSDSLNEKDLIILKSSEPLQDNNRVRLQ
>lcl|NZ_CP026520.1_prot_WP_241688753.1_5562 [locus_tag=PC41400_RS28070] [protein=carbohydrate ABC transporter permease] [protein_id=WP_241688753.1] [location=complement(6208712..6209590)] [gbkey=CDS]
MQGLKLLKKGAVTAMLAVIAGIILIPIVITFTNSLMTEREIGINYGLIGKMNDTAGSEANAFVNLKLLPDLVTLGQYGEV
LIHSPRFLMMFWNSVFLVVPIIAGQTVVAALAAYAFSKLRFRGRETLFLVYVLTMLMPFQVTLVPNYIMADKLGLLHSPG
AIILPGIFAAFGVFMLRQYMLHIPYAYIEAAKMDGAGHWRIFSTIIIPMVKPGVSALAVLLFVDYWNMVEQPLIFLDDPF
KQPLSVYLSRVNQGEQGIAFAASMLYMTPMVLLFLYAETNFIEGIQLSGIKG
>lcl|NZ_CP026520.1_prot_WP_042235074.1_5563 [locus_tag=PC41400_RS28075] [protein=sugar ABC transporter permease] [protein_id=WP_042235074.1] [location=complement(6209605..6210471)] [gbkey=CDS]
MFRRARTDTAAVLMFLAPSALGFMMFYLIPFAMGVYDSFVESTADNRFVGFEHYRELLSSESFRKAARNTASFTLVSVPL
MILLSLSLAVLLNQNIYIRRWLRTAFVLPLVVPAASVVLVWQILLDWNGSVNAWLGYFERARVDWMKTDAARSVVIAVYL
WKNIGYNMILFLAGLQQIPKDYYETARIEGAGPFRQFRSITLVYLTSTLFFVTIMSVINSFKVFRETYLIAGDYPHDSIY
MLQHYMNNMFMSLDIPKLTAAATLMTGSILLLVLVLFALERRFRSFLE
>lcl|NZ_CP026520.1_prot_WP_042235076.1_5564 [locus_tag=PC41400_RS28080] [protein=HAMP domain-containing sensor histidine kinase] [protein_id=WP_042235076.1] [location=complement(6210622..6211755)] [gbkey=CDS]
MPNMIRSFRTKMILLLGISMLLAGLITYLIYKVLQIYYRSQVKFEDPLAYFRLFLHKIGDINFFLLFFIPMVILFFFFFT
RSYAVYFREISDGIHRLASGDFNSRVHVASNDEFEDIAKDINLASEKLQEAVNRGDFAESSKDQLVLNLAHDLRTPLTSV
LGYLDFILRDKELSSEQVRHFTQIAFTKSQRLEKLVDELFEITRMNYGMLKVERNRLDLGELLIQLNEELYPMFEKNQLT
LRLNADQPLHLWGDGELLARVFENLLTNAIRYGSDGQYVDVIAASEGEEAVVRVVNYGDSIPPEELPRIFDMFYTADQAR
NHRTGSTGLGLFITKNIVERHNGTVSVQSNPVRTLFEVRLPKNHSDNETSAPIERTI
>lcl|NZ_CP026520.1_prot_WP_042235078.1_5565 [locus_tag=PC41400_RS28085] [protein=response regulator transcription factor] [protein_id=WP_042235078.1] [location=complement(6211748..6212443)] [gbkey=CDS]
MKRITILIADDEVEIADLIEMHLNKEGYDCVKAHNGHDALQAVQTHSIDLAILDIMMPKMDGYEVTHQIREKHNMPIIFL
SAKTTDLDKISGLVRGADDYITKPFNPMELVARVNAHLRRFLQMNQAAAADKPILESGGLVIYPEQRTVSLYGNLIELTP
KEFDILYLLARYPKKVFSVENIFQQVWGGAYYEGGNTVMVHIRTLRKKLGEDVSKNKLIKTIWGVGYTFNA
>lcl|NZ_CP026520.1_prot_WP_042235081.1_5566 [locus_tag=PC41400_RS28090] [protein=D-alanyl-D-alanine carboxypeptidase family protein] [protein_id=WP_042235081.1] [location=6212659..6213615] [gbkey=CDS]
MNLKTILLGGTAAAIILVMAWNFLDLTRHRLASPVPESSVSEPVISEPAAGKSGKNQLVNESIEGEAAVLMDAETGKLLF
GKNKDLRLYPASTTKIVTALVALEKGGLQDSVTVGDEVRLTTKGESTAGLQEGQTLTLRDLIVAMMLPSGNDAARTVARY
IAEKDSGRKLSAQKSIDYFAGLMNAKASKLGAKKSHFTNPHGLHDPGHYTTAGDMALLSREAMKNKVFSSIVSEPKQAVA
VSSGSLTYVNRNQLLQEGSEFYVQGVNGVKTGFTDEAGYCLVSSATLNGKKFIGVVLHSSSGGVYTDTRSLLEYAKNR
>lcl|NZ_CP026520.1_prot_WP_042235083.1_5567 [locus_tag=PC41400_RS28095] [protein=LytTR family DNA-binding domain-containing protein] [protein_id=WP_042235083.1] [location=6214006..6214761] [gbkey=CDS]
MYRVAICDDEEKQRELVRNILITLSIKTNIEFEIELFGSGEQLSAYYERRETPFHILLLDVEMGGINGIQTARKIRELNN
LDEQIIFLTGYPEYMVESFDVMTFQYLIKPVAPSVLEEKIIKLCQYFQALDKKFMVIKSAYDEVVIKYDDLISIEAAKSL
TIKSKLHFTTVNQTYESKGTISDYALALKDHKFLQIHRSIIINLLHVRKFAKGVVLMSNGTELPIGRSKVKEVKDVYTKF
MIMKVDGYESN
>lcl|NZ_CP026520.1_prot_WP_042235085.1_5568 [locus_tag=PC41400_RS28100] [protein=sensor histidine kinase] [protein_id=WP_042235085.1] [location=6214748..6216040] [gbkey=CDS]
MSQTNFPIFFSMVVVMAVQINFFFNSVFDKTTHKRNRFIYFLIFGLLDYLYLLTPLSPALSSLVSLFLIFTIALSYQVEF
RMKLIFSMLYGVLMSISNTISLYILYTLDHVEFDTLDLVTQPAAQTSYTKLIFLSCMIMFVVIQVIRLIAKRRSFSLPYR
YYVLFLVIPLLSFYHLNILTYKSFYSFLSAIGFLFLNVMIIYIFDTIMDKFQFMHENAQLQQQMDYQDANYEKVVHSFKS
IKRIIHDTNQQFLYIEECIKRNEPAAAVEHIKTTLNKVEGAYHRVNTGNLVIDALVTNTLDIGQANGIRIDTRLNLRSHE
IPIDRYDLCVVMGNMLDNAIEASKHIKIAEDRYILIKIHATESALLIHILNHMEKEDARLHSQKPNPEFHGIGLTNIARI
CDKYGGHMTLETKHKQFNNMVLLPFLTNNP
>lcl|NZ_CP026520.1_prot_WP_042235086.1_5569 [locus_tag=PC41400_RS28105] [protein=serine hydrolase domain-containing protein] [protein_id=WP_042235086.1] [location=6216162..6218213] [gbkey=CDS]
MKKMVSVWMTTMLLTVPVASAFAQTTQDPVKEKAKILASKIVSDYGVSGIQYAIRDHGSVVLSGSAGVKDKAADSPVTKD
TMFGIGSISKVYATAAALKLADTNRIELDKPLTTYLKDFKMADERYKKITPRMLMNHSSGLYGSHYENGFLFDDNDTRTH
DELLAKLRSEHLKADPGEFSVYCNDGFQLLEILVERISGLSYSEFLKTHFSKPLDLNSTKTPRDTFDRTKLAKTYAPEID
QALPVENVNLIGTGGIYSTAEEVSKFGDVLIGNRTDILSEKSAKAMQSHEYRKGVWVPEETNSLNYGLGWDAVKLAPFDD
YGITALSKGGDTQAYHATLTALPEHDISIAVLSSGGSSAYNSVAATNILLEYIKTKGIIKEILPEKTFPPAVKVDMPSEL
QAYSGLYGATDKMINLEIKNGEIDLSAAGGLIPSQKFVYTGNGQFKNKEGNLTVSFDRAKNGKTYLKFNGYLSFPGLGQM
VMATYEYQKLDRNPLDPATRKAWDQRNGKRYYALDEKITSVGYGNQALLTREITVNADLGYASGTKIVNPNKAVNAVTIP
LINGRDSFDLNFYKKNHTEYLTIAGQSYIHEDGVQPFSQGSSAIRSISSNGQAAWYKIDKQSAGRVLTVKAPASGGFAVY
DAKGKLVNFSKISKNYSAVLPEGGLIVFGGHAGDVFKIDLKTK
>lcl|NZ_CP026520.1_prot_WP_042235089.1_5570 [locus_tag=PC41400_RS28110] [protein=TraB/GumN family protein] [protein_id=WP_042235089.1] [location=complement(6218294..6219676)] [gbkey=CDS]
MKSHLWSKWSRKTAAGLLGVTILLGGTNLTVRAEQPPAPTPSISTWSLSTLNEGEKYGIYPVGWYYDGKFQEAIDAVKVK
SLIESTGAKLDQLGFSKKDLPLPVLENKVITRDTVMRAMHGLLARYELPAAFGIGSSSPIEYLQQKGIVQGSSEGLELDQ
PCTAEQAAVLASRLVEFSYDTAGQGAKGVFWKVKHNNNTLYLLGSVHLGIPEMYPLQKNIREAFEASDDLWVEADTLSGD
MSYFANLMVYDDGSKIEDHVSKETYEKLQKVIKKMEIPEQALDGYKPFAVSSNLSMLSLFDQPDQITAASASGIDHYFLT
KALLTGKPVHELEGIKLQADVFNSLTPEAQEKDLNSLLDTLLADGGKEGADSFKQLQLDWSKGDAKGVADGLVRGGLMEG
ELNKKLIGERDKNMAKKIADLLEKDGERTSFVVVGSAHYVVKDMVIDQLKAKGYDVQPVK
>lcl|NZ_CP026520.1_prot_WP_042235092.1_5571 [locus_tag=PC41400_RS28115] [protein=ABC transporter permease] [protein_id=WP_042235092.1] [location=complement(6219774..6221090)] [gbkey=CDS]
MNSMLTVIRFTLLNRLKTKSFKITTLIFVLLISAVVNLPAIIQQFSKEDGKEAVKIGVMANGTDIQQKLTEYFAAAKVGA
STIEIVPLESKGGAEADATLIREKLKNKDVKGVLTAKPAGAGEFPQLLYQTGASGFSMNPGTDETAGLLKAALQTIKTNS
AITQAGIEPAVLAKIQEPVNLTTETVALKEAETPAKSKEELGIAFILVYALLFLLYMSVIGYGSMVSTEITAEKSSRVME
ILISSGSPLKQMFGKIIGICLMGLLQNVVFGVVVAVNLMLPWNGAALQSMDLDLSLIPYGLLGYFLLFYLGGFFIYATIF
AAIGSLVSRSEEVQQAVMPVTLLIIIAFMTAMFGLQNPEAPFVIAMSYVPFFSPLIMFLRIGLGSPAWWEIVISIAIMLG
SIFVLGWLSAKIYRTGVLMYGKRPSWKELRKAMKAFQA
>lcl|NZ_CP026520.1_prot_WP_042235095.1_5572 [locus_tag=PC41400_RS28120] [protein=ATP-binding cassette domain-containing protein] [protein_id=WP_042235095.1] [location=complement(6221083..6221985)] [gbkey=CDS]
MLPLELEHVTKQYGEKTAVNGISLQVKPGEIYGLLGGNGAGKTTTMRMSLGLIYPDGGTIRWNGKGYGDELRRLMGYLPE
ERGLYPKYKVSEQLIYLAELRGIKRKEADRTLKEWLAKFEVPEYYDKRVEELSKGNQQKIQFIASVLHDPSIIIMDETFS
GLDPVNVELLKGTVKELRDNGKAILFSSHRMDHVEELCENICILHRSNTVLQGNLREIKQSFPKERVLLETKEAVPGLET
IEGVTSVEKHLNGYEIRIREAGAAQRILAHAAAETDIYRYQLMEPTLNEIFIAKVGSSHE
>lcl|NZ_CP026520.1_prot_WP_042235097.1_5573 [locus_tag=PC41400_RS28125] [protein=ABC transporter ATP-binding protein] [protein_id=WP_042235097.1] [location=complement(6222025..6222969)] [gbkey=CDS]
MNREAERYDDCAIRLSGVEKMLGDMRLGPLELEVETGTVVALVGQNGSGKSTLFRLLMNLTKADSGKVELLGRPYGEDDT
EADIKRRIGYVPEESNWEDLGCTTIGELHDFVCRWYPTWNETLYRSMTEEFELSGKLKLKTLSKGMKRKLSYILAAAYEP
ELLLLDEATSGLDPFATKRMIEHLAKFMEREGRSIFFSTHVMEEVRRLADYIVYINRGRLIGVYEKDQLLDDWKCVWVRE
TPQPSSGPLPGVVSVGNGEHGGLIPIISRSSSATEAALASQGNAPVRTTAMDLEDILELLAQEKIEKKAENKER
>lcl|NZ_CP026520.1_prot_PC41400_RS28130_5574 [locus_tag=PC41400_RS28130] [protein=GntR family transcriptional regulator] [pseudo=true] [partial=3'] [location=complement(<6223103..6223477)] [gbkey=CDS]
MHIPIQISPDNPEPLYHQIQVQLKELILSGHLPEGTLLPSIREFAKDLSCSVITIRRVYQDLENEGILRTRQGTGTFVAK
VGTVTREQYRKEAVVEAMKDALRSGLRVGCTIEELKQLFEEALLE
>lcl|NZ_CP026520.1_prot_WP_042235101.1_5575 [locus_tag=PC41400_RS28135] [protein=ABC-2 transporter permease] [protein_id=WP_042235101.1] [location=complement(6223529..6224266)] [gbkey=CDS]
MSVWSNVWWLAKQDLKMVRYGYIWSLLYYLNAMFVLFVFMKPEISMHGEVNGFLTVVFNLFVLSIIQVVGYPIDTRNMNY
WKNDAYTKRILQLKMLPVTNKEIVTSRFLVYILYSSFFILLLHVGIRVMMSLQWVEPVGWGTWLGMIFFWWGAVLAVGSF
YLYGELSMRGKVFFRFSLLMLLATLILSILLWWTGNYMVWPLAISSIERYGVIVPACSLIIGATALGFSWKILGRKIGHR
DLFAS
>lcl|NZ_CP026520.1_prot_WP_009671726.1_5576 [locus_tag=PC41400_RS28140] [protein=DUF420 domain-containing protein] [protein_id=WP_009671726.1] [location=complement(6224627..6225082)] [gbkey=CDS]
MHVLPTISTMFIVISAIFVAFGWYQIRKGNRETHTKLMIAGAISALIFFTIYMSRTIFQGNTSFDGPESLKLAYHIFLFF
HITLATVGAVFGIVTLILAFKKNWIRHRKVGPITATIWMFVAPTGVVVYILLYILFPGGETKPLIDAIFGL
>lcl|NZ_CP026520.1_prot_WP_042235104.1_5577 [locus_tag=PC41400_RS28145] [protein=cytochrome c oxidase assembly protein] [protein_id=WP_042235104.1] [location=complement(6225137..6225961)] [gbkey=CDS]
MGFFELWNPVILLFVVVVGLVYRSLTSPSRAAAAGEEPVSSKVKLSFYTGIALFYIGQGSPINYLGHHYLFSVHMLQQSL
LYLIMPIFILNGLPAWLIRPLFKGTVMNRLLRFFTHPLISVLAFNLLFSIYHLPLIMDTLMENEWMLLGYHGVLLFTAFA
MWFPVFTPLPEHEKLSSLMKLAYIAVNGILLTPACALIIFASEPIYAMYANVVIPFNALPLLDDQQLGGVLMKVIQEIVY
GSALAMIFFKWYRTERAQEDESDDLQENNGLTTA
>lcl|NZ_CP026520.1_prot_WP_042235106.1_5578 [locus_tag=PC41400_RS28150] [protein=cytochrome C oxidase subunit IV family protein] [protein_id=WP_042235106.1] [location=complement(6226063..6226374)] [gbkey=CDS]
MSTNETTSPSKRYKKESPANHYFSYIASILLTMIAFALVMYGGLDKSFLYAFLIFMALIQITLQLAYWMHMKEKGHLFPI
VGIIFGGIVALTAVAAGVYWTWW
>lcl|NZ_CP026520.1_prot_WP_042235108.1_5579 [locus_tag=PC41400_RS28155] [protein=cytochrome (ubi)quinol oxidase subunit III] [protein_id=WP_042235108.1] [location=complement(6226376..6226996)] [gbkey=CDS]
MSTHTHDTGTLPLEPEKATLEGKNKVLGFWLFLGGEMVLFGTLFATYIALRNQIPDGPTPMELFKLSTVAWATFILLTSS
LTSVFAIMAMHKNNVKGLLFWLAITVLLGLGFLGLEVYEFYEYVHEGHGFTTSAFSTSFYTLVGFHGGHVAFGVLWITLL
IIQGTRKGLTVVTAPKFYVASLYWHFIDVVWVFIFTVVYLMGKVGP
>lcl|NZ_CP026520.1_prot_WP_042235125.1_5580 [gene=ctaD] [locus_tag=PC41400_RS28160] [protein=cytochrome c oxidase subunit I] [protein_id=WP_042235125.1] [location=complement(6226993..6228816)] [gbkey=CDS]
MDWLTTVDHKKIGILYYMAGGLFFLIGGLEAILIRIQLMNPELKIFIGDTFNQLTTMHGTTMIFFAAMPMIFGFMNAIIP
LQIGARDVAFPFLNSLGFWLFLFGGILLNLSWFMGGGANAGWTAYAPLSSIVDSTKAITVGGNITGVDFYVLGLQIAGLG
TLLGGINFLVTIINMRAPGMTYMRMPMFTWASFITSALILFAFPAITVGLVELMFDRIFDGAFFDPARGGSTVLWEHLFW
IFGHPEVYILILPAFGIISEIVSTFSSKRLFGYSSMVFATALIGFLGFMVWVHHMFTVGLGPVANSIFSIATMAIAVPTG
IKIFNWLFTLWGGQIRFTTSSLFATAFIPTFVMGGVTGVMLSVPAADFQYHDTYFVVAHFHYVIVGGLILGLFAGSYYWW
PKMFGRVLNETLGKIHFWTFFIGFHLTFFPQHFLGLMGMPRRVFTYDPGLTAGNLVSTVGAFLMGIGTIVFIINMIATNR
KPQDAPDDPWDGRTLEWTIPSPAPEYNFKQTPLVRGLDALWKEKMAGNKEMTPAEPVGSIHMPSPSILPLFMALGLFISA
FGFMYHIYVICIVGIGITLLCMLLRSVFDDHGWHIEPEELNDKGVKR
>lcl|NZ_CP026520.1_prot_WP_042235110.1_5581 [gene=coxB] [locus_tag=PC41400_RS28165] [protein=cytochrome c oxidase subunit II] [protein_id=WP_042235110.1] [location=complement(6228912..6229910)] [gbkey=CDS]
MSRWKKNWRLVPLFAVMTFLLAGCGDPTLSALRPRGPVASEQLFLMKISIAIMLLVVVVVFALYLFVIIRFRKRKGQEGI
PKQVEGNHVLEIVWTVIPILLLIVLAVPTVYYTFKHTTNYVQDKDAIHVKVTAHQFWWQFEYPELGISTAQDLVIPTGKN
IALELTSADVTHSFWVPSLGGKMDTNTNMTNVYYLKADEVDVFKGKCAELCGASHSLMDFKVKSLAQGDFDAWVTNMQTP
AAVPADAKQGEELFKQKCLQCHAIDPKGLGFGPNLNGYADREMLAGVLEHNDENLKKWIKDPQAEKPGTLMPKVPLEDNQ
VNEIVKYLNSLK
>lcl|NZ_CP026520.1_prot_WP_042235112.1_5582 [locus_tag=PC41400_RS28170] [protein=hypothetical protein] [protein_id=WP_042235112.1] [location=6230208..6230882] [gbkey=CDS]
MKKIMRRFIPALSAAGFIIVTVSGCSNTYPKETSYGHPKQRIEGQGQSRAYQTQQQVSAVSHNNTFFAYDHELSTQVAAM
HGVMGAIVMRTDRNAYAAVMVDNSASGTKAGRSETKTHGPGDKKNQIPQPYYENLYTNELASDFIGYVTVQDHKNLSHTF
KQEIALKIRQLDPTVHDVYISANREFINQLNHYKLDALSGASLKPYVNEFNQGATRIFGTKPQP
>lcl|NZ_CP026520.1_prot_WP_129112549.1_5583 [locus_tag=PC41400_RS28175] [protein=DEAD/DEAH box helicase] [protein_id=WP_129112549.1] [location=6231054..6232931] [gbkey=CDS]
MSFPFIDLGVRPELSKLLARSGVSEPTPIQQQALPVVFRGGDVIAQAQTGTGKTLAFLLPMLERTNPTRAEVQGLILTPT
RELAIQIAAEAAKLAPALGINVLAAYGGQDVEKQLRKLSSTGGVHLIVATPGRLLDHLRRGSVALYKISMLVLDEADQML
HMGFLNEVEEIIAQTPQRRQTMLFSATMPDPIRNLAKRYMVNPEEIRVKAKQVTLTEIDQSVVRVTDRGKLDALCRAIDT
DKPYLAMIFCRTKRRAASLNADLQERGYKSDELHGDLSQAKREQVMKRFRDASIQLLVATDVAARGIDVEGVTHVFNYDI
PHDAQSYIHRIGRTGRAGQTGVAVTFASPRDQVYVDQIEQGIRQQLRRQDAPRRERGEETDERRERGGRSPRGERSFGGS
DRAKPAGKGGGRSRGGDGAGRSEGRWEHRADGPRGGSGGRGRGRADARDAGGPRSGGAGRGTRGDGGFRGDNAERRDSAD
RGGRGRSGAERGGFGGARGGSGRDDFGGARGGSAGRGPRSESARDSGRGARDRDSVAGPFGEAGWGVSARQREETEARNS
RGRGGADSRGPRGGGSGRGDFGGSRSGGAGRGSRGGGSNGRGAGPRGGGSRSGGPGPRGGSKRGR
>lcl|NZ_CP026520.1_prot_WP_042234239.1_5584 [locus_tag=PC41400_RS28180] [protein=conserved virulence factor C family protein] [protein_id=WP_042234239.1] [location=6233291..6234454] [gbkey=CDS]
MNLVSIEPTPSPNTMKLNLDKALPKGRSLTFTRKQRDNAPGYIQQLLDIPGVTSVFQVADFIALDRSPKGDWKQILTQAR
ELLGSGGSQASLPSEDEAGSGFGGVKVLLQTFRGIPMQIRVNAGGEEARAALPERFTQAAMKATENSPHLLKERKLQELD
VRYGELQDVLNEVVQEVDASYDDNRLNLLTEQAAKGLSEADYTEAQRPAELTEEEIGRLLDDPDWKKRYAALQSVKATPQ
TLPLLAKALGDEKTSVRRLATVYLGDIKEPEVLPYLFQALEDRTPAVRRTAGDTLSDLGDPAAVPAMSKALKDSNKLVRW
RAARFLYEVGDDSCLEALREAVDDPEFEISMQVQMAIARIEGGHAAEGSVWQQMTRSRQGTDAPKQD
>lcl|NZ_CP026520.1_prot_WP_042234241.1_5585 [locus_tag=PC41400_RS28185] [protein=M48 family metallopeptidase] [protein_id=WP_042234241.1] [location=complement(6234593..6235852)] [gbkey=CDS]
MKNWKVVLILGFAVYAAAVGFYFGRGDIYSLPSGSAGTSIDPHTFMSEGEIGRAEVLSRIRSIVYFLQTPLQLVLLILLV
GTAIRFRDRIEGWTRRYLLRTGLFVFLFLLVVYVLMLPFDYFMLRVERHYGVSNVSLGTWLVDHAKNLGIEWLTTTFVLL
LVLALMKRSPRKWWLWLWTFSVPLLLFLQFIQPVIIEPLYNEFAPLRAGELKKELLHLASQAGIAADDVYEVNMSERTNQ
LNAYVSGIGSNARVVLWDTLLQKMDTREILVVMAHEMGHYAERHVFWGSGLGLVMSFGLLWGGSRIYEIAVRNWGRAKGL
RGPRDMAGLPLLLAIVTLLTLAVTPINNAASRLMETRADSYAMKMTGDGQAGVSAFQKLAAANLSPVTQPALLQWFLGSH
PTIEERIRYFGEFAAQKAR
>lcl|NZ_CP026520.1_prot_WP_042234244.1_5586 [locus_tag=PC41400_RS28190] [protein=MBL fold metallo-hydrolase] [protein_id=WP_042234244.1] [location=complement(6235958..6236926)] [gbkey=CDS]
MNRKRYTNMDPAAGLSTFSDLRKWRKERSGKVKDLSFVVKTSPDKDPGYLKANRSETTITWIGHSTFLLQKNGINLVTDP
VWASRMGFDKRLAEPGLALGELPPVDAVLLSHGHYDHLHIGSLRALPGNPKLLVPEGLGKTLRRKGFKDIEELSWWGSYE
FAGLRFHFVPAQHWTRRSLTDTNTSHWGGWIVEDAGKPGTDPDSLYFAGDSGYFQGFRMIGERFRIGCALMPIGAYEPEW
FMSTQHVSPEEAVQAFLDTQAEIFVPMHYGAFRLADDTPQEALDRLRLNWEHRKLDPQRLQVLLHGQTLKTASGGTANKI
SR
>lcl|NZ_CP026520.1_prot_WP_042234247.1_5587 [locus_tag=PC41400_RS28195] [protein=Gfo/Idh/MocA family oxidoreductase] [protein_id=WP_042234247.1] [location=complement(6237221..6238228)] [gbkey=CDS]
MSSVNRKIRWGILGCASIALRAIVPAIQASSTGELAAIASRDGSKAREAARRHGIPAAYGSYEELLADPTLEAVYIPLPN
HLHFEWTIKAARAGKHVLCEKPLALTAAQAQEMVAACREAGVLLAEAFMYRHHPRYDRIREILAAGELGVIRAMRASFTF
DNSADTGNVRCRKEWGGGSLYDIGCYTLNAARYLLGGEPEAVTVQAFFSPGHGGVDMMASGLVEFPGGVALTFDCGMWAA
SRNTIEIVGTAGTIEVPSAFVAPEGRGADFTVTLNGTCRKEPTPPVNQYAVQIDDFGRAVLGREPYRFPAEDAINGMKVL
EACLLSAETRARISL
>lcl|NZ_CP026520.1_prot_WP_042234251.1_5588 [locus_tag=PC41400_RS28200] [protein=DUF2238 domain-containing protein] [protein_id=WP_042234251.1] [location=6238380..6239006] [gbkey=CDS]
MKSSRVPVRSYYALLGSLAAVFIWSGIRPHDYFTWLLEVTPGLILIAVLIGIYRRTPLTLLTYTLVWAHVILLFVGGHYT
YAEVPLFNWLRDTFDLERNYYDRLGHFMQGFVPAILTREILIRKIPLQPGGWLVFLTLSVCLGFSAFYELLEFTVAKLTG
TAAESFLGTQGDVWDTQWDMMCALIGAALSLLLLGKSHNRAMRLPGSF
>lcl|NZ_CP026520.1_prot_WP_042234253.1_5589 [locus_tag=PC41400_RS28205] [protein=hypothetical protein] [protein_id=WP_042234253.1] [location=6239055..6239366] [gbkey=CDS]
MNRNQQEMEQRIIENYQRDERMMILIFAQWCINHGLDPAGLYAQAYPQQGNNTELQQALELTVSKEEAGEIPDDTLLGVL
SMFGNEELAFVVTEEIAKRPPRK
>lcl|NZ_CP026520.1_prot_WP_042234255.1_5590 [locus_tag=PC41400_RS28210] [protein=hypothetical protein] [protein_id=WP_042234255.1] [location=6239521..6239745] [gbkey=CDS]
MQPITSKELDYLADSLTNEDLLIKQAANAIAHAQNPAIRQIAEHLIQSHQQHYTQILSALQQHAQLAPNQPQQQ
>lcl|NZ_CP026520.1_prot_WP_042234259.1_5591 [locus_tag=PC41400_RS28215] [protein=spore coat protein] [protein_id=WP_042234259.1] [location=6239779..6240327] [gbkey=CDS]
MNQQPNNQNRQLSYQQQQQHDQRQQQLEQLFQQQSAYPYSQGFQQNQQQNQQQNLQSGSILSDKDLLSTTLADHKRVIRE
YATAATESNCPAIRQLFNQLTLDTLHLQGELYTFMKQNNMYSPPSPALRQDIDKQIQHSQQSGQQTQQLVAPYLSFGGPH
QGYSYIQPSYNQAGQAQQHYTM
>lcl|NZ_CP026520.1_prot_WP_042234261.1_5592 [locus_tag=PC41400_RS28220] [protein=hypothetical protein] [protein_id=WP_042234261.1] [location=6240678..6240986] [gbkey=CDS]
MTALWGTLTLLIFGFLVVTVFIKSIQGARDQYGVSSFDYEEEEHTSPAHHEWMAEPGPDSASVKEWATAEVGSARTDGIH
SRQPVQLRAGSEAASEQKQPNG
>lcl|NZ_CP026520.1_prot_WP_042234263.1_5593 [gene=thiC] [locus_tag=PC41400_RS28225] [protein=phosphomethylpyrimidine synthase ThiC] [protein_id=WP_042234263.1] [location=complement(6241262..6243037)] [gbkey=CDS]
MSKSTEIQEPNVSAFPGSRKVYVEGSRPDVRVAMREIQLSPSSGRFGEEENAPVRVYDTSGLYTDTEEATDIRRGLPANR
YGWIAERGDVEAYEGRTVKPEDNGLRSEEALANTEVFPGLNRKPFRAKPGGNVTQMHYARKGIVTPEMEYIAIRENVAPE
YVRDEVARGRAVIPSNINHPESEPMIIGRNFLVKVNANIGNSAVASSIEEEVEKMTWATRWGADTVMDLSTGKNIHTTRE
WIVRNSPVPIGTVPIYQALEKVGGKAEELTWEIYRDTLIEQAEQGVDYFTIHAGVLLRYIPLTAKRTTGIVSRGGSIMAA
WCLAHHQENFLYTHFEDICEIMKAYDISFSLGDGLRPGSIADANDEAQFAELDTLGELTRIAWKHDVQVIIEGPGHVPMH
LIKENVDRQMEVCQEAPFYTLGPLTTDIAPGYDHITSAIGAAMIGSFGTAMLCYVTPKEHLGLPNKNDVREGVIAYKIAA
HAADLAKGHPGAQDRDNALSKARFEFRWRDQFNLSLDPERAMEYHDETLPAEGAKTAHFCSMCGPKFCSMRITQDIREYA
RENGLGEQEAAEAGMADKAAEFRREGSVIYK
>lcl|NZ_CP026520.1_prot_WP_129112550.1_5594 [gene=thiE] [locus_tag=PC41400_RS28230] [protein=thiamine phosphate synthase] [protein_id=WP_129112550.1] [location=complement(6243195..6243884)] [gbkey=CDS]
MAAMTKTAAEARYTGNLREMLKVYLVMGSPNCGGRDPAAVLQAAIEGGVTLFQFREKGPGALQGQERTELGNRLLGVCRQ
AGIPFIVNDDLRLAEELEADGIHVGQEDLAQWDREGLRQRFKGRILGISAHDPAEAREAVRCGADYLGAGPMYATATKPD
ARAVQGPGMIAAMRAGGISLPLVGIGGITPEGAASVLAAGADGVAVISAISGADDPREAARRFSRPFRR
>lcl|NZ_CP026520.1_prot_WP_042234265.1_5595 [gene=thiD] [locus_tag=PC41400_RS28235] [protein=bifunctional hydroxymethylpyrimidine kinase/phosphomethylpyrimidine kinase] [protein_id=WP_042234265.1] [location=complement(6243872..6244789)] [gbkey=CDS]
MNISLNKQQHPPEPSRQAPLVPAAATIAGSDSGGGAGIQADLKTFQELGVFGASALTAVTAQNTLGVHGVHPIPPSMVTA
QIDAVAEDIGPAAWKTGMLFSAEIIEAVADRASFYKWSRLVVDPVMIAKGGASLLQENAVQALRSLLLPLAQVVTPNIPE
AEALTGMRIEAMADRKEAARRIHAWGVKYVVLKGGHGGNRAQSLDLLYDGRVFTEFAAPWIETRHTHGTGCTFAAALTAG
LASGLPMDEGVLLAKRFIQAALTNPLNLGAGHGPTNHWGYNRQGTAGVFFQTAPAAAGEGEGWLR
>lcl|NZ_CP026520.1_prot_WP_042234267.1_5596 [locus_tag=PC41400_RS28240] [protein=thiazole synthase] [protein_id=WP_042234267.1] [location=complement(6245020..6245787)] [gbkey=CDS]
MLKIGPYEFNSRLLLGTGKFPDFTIQKKAVEVSEAQILTFAVRRMNIYDPEQPNFLSMLDVSKFKLLPNTAGAKTADEAV
RIAKLAKASGLCDMIKVEVIGCDKTLLPDPVETLKATEMLLAEGFLVLPYTSDDVVLARRLQEMGVHAIMPGASPIGTGQ
GIINPLNLSFIIEEARVPVIVDAGIGAPSDAALAMEMGADGVLLNTAVSGAADPVRMAYAMKLALEAGRAGYEAGRIPRK
RYATASSPLEGVSVG
>lcl|NZ_CP026520.1_prot_WP_042234269.1_5597 [gene=thiS] [locus_tag=PC41400_RS28245] [protein=sulfur carrier protein ThiS] [protein_id=WP_042234269.1] [location=complement(6245792..6245995)] [gbkey=CDS]
MKLHVNGDDITVPETVHTVEGLIRHLGLKRNLLIVETNGHILQAEEHADAPVADGDRIEIVHFVGGG
>lcl|NZ_CP026520.1_prot_WP_042234271.1_5598 [gene=thiO] [locus_tag=PC41400_RS28250] [protein=glycine oxidase ThiO] [protein_id=WP_042234271.1] [location=complement(6246074..6247234)] [gbkey=CDS]
MNRDTDIVVIGGGVIGCSIAYQLAKRGVRVTLLEKDRIAGQASGAAAGMLGAQSEMHEAGPMFDFARRSRDLFPDLAEEL
KELTGLDIGLTHRGMLKAAWTDAEAAALRADGRFHQEAGGRAEWLSPCETGWLEPGLSKELRGALHLPDEGHVLAPELSA
ALAAGAARLGARIMEHADVTGLLTESGRVIGVQTREGKIRSAGTVLASSLWSAQILAPLGVELPVYPVKGECFSLTVRQP
LLARTIASDGCYIVPKPGGRIIVGATAQPGSFDRSPRLGSLLSLAAQAQRLVPALAEAEWERAWAGLRPQTPDGMPYIGE
ITACKGLFAALGHYRNGILLSQATGVLAASVLTGGAAAAEAALPGAGELLEAFRPDRKAAAAAVRQ
>lcl|NZ_CP026520.1_prot_WP_042234273.1_5599 [locus_tag=PC41400_RS28255] [protein=thiamine phosphate synthase] [protein_id=WP_042234273.1] [location=complement(6247231..6247887)] [gbkey=CDS]
MIPYPAGPELHAISTGKQKPEELIRIATAIHPYVTAIHVREKKQSPRELWTLLSELAACGVPPGKLVVNDRADAARLSGA
AGVQLPGSGLPASAVKKGFPGLRVGVSVHGPEEGGRAAKEGADYVIFGHVFATGSKPGAAPRGAEALQETVLACGGLPVI
AIGGILPENCAEVIRAGAAGVAVMSGIWEAENPAEAAEHYRHELELALQPEKTGGELR
>lcl|NZ_CP026520.1_prot_WP_042234275.1_5600 [locus_tag=PC41400_RS28260] [protein=TraR/DksA C4-type zinc finger protein] [protein_id=WP_042234275.1] [location=6248567..6249136] [gbkey=CDS]
MAALTEEERNKLEQRLYKEKGELEHVLESDREEDSLRDSTGELSSYDNHPADIATEAFERGRDQAVDDNMEQQLTDVKHA
IGRLESGSYGVCEVCKEPIPFERLEAIPWASTCVEHSQTRTEDDSVDRPVEEEILEPILESPNDDDRGRTEIDDTDAWKT
VEEYGNSSDTRRTGNAQPKEPADDCEDEK
>lcl|NZ_CP026520.1_prot_WP_042234277.1_5601 [locus_tag=PC41400_RS28265] [protein=zinc ribbon domain-containing protein] [protein_id=WP_042234277.1] [location=complement(6249257..6249739)] [gbkey=CDS]
MSGFIGKIKQGAAEAGKKAQQTIEVNRLKLQIAAKQRDMQKLYTQIGEFVYEAGKQKQPYPETQVAAVTAELQAIEDEIK
AMQNRIYSLYDEKSCECGHVVAVDTRFCPNCGRKFEPIQEQADFTAIMELKPFCPQCKEDIEPGMQFCTTCGYRLTEEES
>lcl|NZ_CP026520.1_prot_WP_042234279.1_5602 [locus_tag=PC41400_RS28270] [protein=DUF4127 family protein] [protein_id=WP_042234279.1] [location=complement(6250252..6251868)] [gbkey=CDS]
MTKIVFVPLDERPCNLEYPQRLAEMTDLELTAPPMSILGRKKTPGDTEALGNWLLREAQGADYALVSLDMLVYGGIVPSR
LHYLTREDALKRLEVLRELKRNKPDLRIHAFNLIMRVPAYSSSEEEPDYYADYGRELFLYGWYTDKKEQEALTDEEERIF
AGILDKIPQDVLEDFTGRREINAFINRTALELAEEGVIDYLIIPLDDNSQYGFSPLEQRALAILSEELDLTDRVAIYPGA
DEIGCTLFARIFCESKGYLPKLFVRYSSTRGPFCIPKYEDRSLNESIKAHLTSAGAFMADSSQGADAVLMVNSPAAGESD
MAETSQAWSQRHRSYFSEVNLREFAQAMRVYLGEGRTVALADVAVCNGGDTVLLKLLARQGQLKGLSAYAGWNTSGNTLG
TVIAHAVIEAYYAGARKSAGGTGAEAEPEGRLRSEAQKRSSRAFYFYRLVEDWGYQALVRKDVKANVLPAFGAGYFELAG
KQPEIEAVIAEKLRSFASERLAGLTSGQIRIANVHLPWNRMFEVGFDLDLVDGSSPQP
>lcl|NZ_CP026520.1_prot_WP_042234281.1_5603 [locus_tag=PC41400_RS28275] [protein=N-acetylmannosamine-6-phosphate 2-epimerase] [protein_id=WP_042234281.1] [location=complement(6251931..6252623)] [gbkey=CDS]
MEKHSLLEAVKGGIIVSCQALPDEPLFGSEMMVRMALAAEAGGAVGIRANTPVDIAAIKSRVKLPLVGLWKRDYEGSDVY
ITPTLEEVEAVWRAGADIIALDATMRPRPKGETLKEIVSEIKKRAGVLLMADISTVEEGLHAQEAGFDILSTTMSGYTPY
SPQQEGPDFTLIREMAARADIPVFAEGRIHTREDAIECFRSGAHTVVIGGAITRPQQITKRFVDFVRIHV
>lcl|NZ_CP026520.1_prot_WP_042234283.1_5604 [locus_tag=PC41400_RS28280] [protein=HAMP domain-containing protein] [protein_id=WP_042234283.1] [location=complement(6252953..6258730)] [gbkey=CDS]
MSEQEQRSNDSLDVLDEKQLLQVLLAFKKGDFSVRMPYDLNGMPGKIADTLNDILDMQENVVQEVQTVERVVGKEGNTNR
RFMCRSTGGSWNAFNDSLNNLISDLIQPTSEMVRVINAVAQGDLSQKVELEFEGRALTGEFLRTANNINRMVNQLGTFAA
EVTRVAREVGTEGILGGQAEVTDVSGTWRDLTESVNNMASNLTDQVRNIADVTTAVAKGDLSRKITVNAKGEFLELKNTI
NTMVDQLSTFASEVTRVAREVGTEGKLGGQADVKGVSGTWYDLTESVNDMASNLTNQVRNIAVVTTAVANGDLSKKITEE
AQGEILELKNTINTMVDQLSTFGSEVTRVAREVGTEGILGGQADVKGVSGTWRDLTESVNYMATNLTNQVRNIAEVTTAV
ATGDLSKKIAVDAKGEILQLKSTINTMVDQLSTFASEVTRVAREVSNDGKLGGRADVKGVSGTWKDLTDSVNYMASTLTD
QVRNIAEVTTAVAKGDLSKQITVNATGEILELKNTINTMVDQLSTFASEVTRVAREVGTEGKLGGQANVKGVSGTWKDLT
ESVNDMASNLTNQVRNIAVVTTAVANGDLSKKITEDVQGEILELKNTINTMVDQLSTFASEVTRVAREVGTDGRLGGQAQ
VKGVAGTWKDLTESVNDMASNLTDQVRNIADVTTAVAKGDLSLKITVDVKGEMLQLKSTINTMVDQLSNFASEVTRVARE
VGTEGKLGGQADVKGVSGTWKDLTDTVNYMTGNLTSQVRNIAAVTTAVANGDLSKKITIDVKGELLELKETINTMVDQLS
SFASEVTRVAREVGTDGKLGGQAHVNGVGGIWKDLTDNVNIMATNLTNQVRGIAKVVTAVANGNLKQKLMVDAAGEIYDL
SETINNMIDTLDTFADQVTTVAREVGAEGKLGGQAHVPGAAGTWRDLTDNVNYMASTLTTQVRQITNVATAVTKGDLSRT
IDVIAAGEVGTLKDNINEMIRNLKETTRINTEQDWLKTNLARFSRLLQGQRDLYEVCRMIMSELAPLVSMQHGVFYMNEA
NGEEQVLRMFSSYAYQQRKHLSNEFRMGEGLVGQCLIEKQRILLTNVPEDYVMIGSGLGEAAPLNIVLLPIMFEDQVLAV
MELASFRRFTEIDIAFLDQLTESIGIVLNTMQANMRTEELLSQSQSLAEELKKQQEELQHTNDELEDKAKLLVIQKAEVE
SKNNEVEQAKRFLEEKAEQLALTSRYKSEFLANMSHELRTPLNSLLLLAEQLSENPDHNLTDDQVKFARTIHDSGLDLLN
LINDILDLSKIESGTITPDFTEMKLTEITDNMERTFRHIAQDKKIDFKIKAEKALPESIISDSKRLQQILKNLLSNAFKF
TEKGEVVLLMRRASSGWSQDNELLNRARMVLAFSVGDTGIGISEDKQQIIFEAFQQADGSTNREYGGTGLGLAISREIAG
MLGGEITLFSVVNVGSTFTLFLPVDLDPAMLEAAAAYVPEVIDVPAAQPKKYEYDRFDTTQLVDDRANIEAGDRVFLIVE
DDMKFNEILLDMLHRKGLKAIITPKGNKAVALAKRYNPVAITLDLRLEDMNGSSVLDQLKDDIELRHIPVTVMTIEDDAT
ELLQKGVFDFLCKPIKNEDLENAIDKLNAFGTKPFQQLLVGMSDSERRRQTVDLVKNSDIKVSATDTGRKALNQLKSKDF
DCVLVDTDLPDMGPAQFVREMQKNAKNKYKPVLINRTEKLTKEEEDDLEELRSIAVIKEVKSPIQIVDETMLFLHRKLEN
LNDSQRDILIKKHQSNEMIDSKKVLIVDDDIRNIFALTTILERQNMKVISAENGHDAIEILEQMPDINIVLMDIMMPVMD
GYATMREIRRRPQFQDLPIIALTAKAMKGDREQCLEAGASDYITKPVNSAQLLAMLRDWFLAQDGLQPSPAELPGSEPFG
WMKGQ
>lcl|NZ_CP026520.1_prot_WP_042234286.1_5605 [locus_tag=PC41400_RS28285] [protein=ROK family protein] [protein_id=WP_042234286.1] [location=6259422..6260369] [gbkey=CDS]
MSSYAAIGIDIGGTHIKSGLILPDGKVLASHKTETEASAGKAGLMAKLAGLLQTYSEEAAAAGVKLAGTGIGSAGYINRE
GTVAFATDNLPGWSGTTLRADIEKLSGLPAAVDNDVNVIALGEQWLGAGSGWDTFLCVALGTGVGGCYIREGSPLHGRDG
FAGAYGHMVVAFGGEPCTCGHRGCWEQYASVTALKRLIRERGGQTLEEASPPELFARARAGDETAMGIVRLYAEYVATGL
SGLIHAFNPPAIVVGGAVTEQGDFLFDLIRRHVLDMTMPAYTAAPVPIVPAELGGTAGMFGAAKLLFDRLKVEQP
>lcl|NZ_CP026520.1_prot_WP_241688754.1_5606 [locus_tag=PC41400_RS28290] [protein=phosphatase PAP2 family protein] [protein_id=WP_241688754.1] [location=complement(6260674..6261330)] [gbkey=CDS]
MNTKLNVLLTRSFLVCLLLAVCFGTVALLVSRRQIAWFDETIISFVQGAESPGLTAAAGLLSEIGSSAVVPFITIAAALY
LYFVLKHRAEVILLIAAIGGSAILNTILKKIFQRDRPAIHIIMEETGFSFPSGHSMAVVGLYGILVYLLWKHISSAAGRA
VLVTLATAMIVAMGLARIYLGVHYPSDVLGGYLASGFWLGVCIWTYRSYRSRRQSAVR
>lcl|NZ_CP026520.1_prot_WP_042234291.1_5607 [locus_tag=PC41400_RS28295] [protein=metal-sensitive transcriptional regulator] [protein_id=WP_042234291.1] [location=complement(6261531..6261791)] [gbkey=CDS]
MNYDESIVRRLKRVEGQVRGILKMMEEEKDCKDVVAQLSAVRSAADKAMAYIVATNLEHCIRQEEEKGKDTGKLVQEAVE
LLIKSR
>lcl|NZ_CP026520.1_prot_WP_042234293.1_5608 [locus_tag=PC41400_RS28300] [protein=rhodanese-like domain-containing protein] [protein_id=WP_042234293.1] [location=complement(6261831..6262241)] [gbkey=CDS]
MIMWEIIAGMMILVLVLSWLPVRGLSYLSQEKARELMASREEGRIKVLDVRDASDYEEAAGFPSINMSIGRLPYLWKDEF
SPEDRVLLVGDSNRHIKKAARFLKRRGFKGLYAMQLRSELRALGTCREKTCGAGCP
>lcl|NZ_CP026520.1_prot_WP_042234296.1_5609 [locus_tag=PC41400_RS28305] [protein=rhodanese-like domain-containing protein] [protein_id=WP_042234296.1] [location=complement(6262321..6262641)] [gbkey=CDS]
MAFQIPKSCTSQDLARRIHDGEDLYILDVREDEEWADGHIEQAVHIPLGQLMQRLGELDRDREITVVCRSGNRSGLACEL
LTEQNFDVVNMTGGMNNWTDRISFGR
>lcl|NZ_CP026520.1_prot_WP_042234298.1_5610 [locus_tag=PC41400_RS28310] [protein=DinB family protein] [protein_id=WP_042234298.1] [location=complement(6263028..6263585)] [gbkey=CDS]
MATDSQLQEREKAALLSGYAEQQYDYHVWANGKVFAGLEALPEELSFEQIQSIFPSVADALNHIYMMDEMWLVIMKGGGY
EEARKVVTDLYAQGRGDSLDELRKRFALAEEKYRAFFAEGEGAAAETAPSHPQFGRLETHVYALVQHVVNHGTYHRGNIS
AMLRQLGHPGTPTDYIGFLFERGLR
>lcl|NZ_CP026520.1_prot_WP_042234301.1_5611 [locus_tag=PC41400_RS28315] [protein=WYL domain-containing protein] [protein_id=WP_042234301.1] [location=complement(6263816..6264781)] [gbkey=CDS]
MSKADNMLAVLWHLKSGRKMTARELAETLEINIRTVYRYIDALCASGVPILADSGHNGGYSLLPNFREAPLFFNAAEQKA
LVHAAIFAEEAGYPFGDALNSAVGKLKQYTNPEQLRFIERHEAGFDVIHPPTDSSLEGYLQMLEIAVADGYTVRMEYSKG
YAAPVQSRDVDPYGLVHWRSKWYLVAFCRLRGELRSFRVDRIRVLSQTEDRFDRPQGFWARHFFMNNLLPDTSQEDQLVS
VRISGSPESIAELCDHWLFGRAIVERTTDQVHMKMEEKAIRSYVPYFLMIHGRSIRILEPPLLKEQLIAATSGLLEFYKQ
L
>lcl|NZ_CP026520.1_prot_WP_042234302.1_5612 [locus_tag=PC41400_RS28320] [protein=MDR family MFS transporter] [protein_id=WP_042234302.1] [location=6265258..6266781] [gbkey=CDS]
MAPAKTKHLGIVIAGLLLGILMASMDNTIVSTAMGTIVGELGGLDKFVWVTSAYMVAEMAGMPIFGKLSDMYGRKRFFVF
GIIVFMIGSILCGTATSIVQLSVYRAIQGIGGGALVPIAFTIMFDTVPVEQRGKLGGLFGAVFGMSSIFGPLLGAYITDY
IHWEWIFYINLPLGLIALVFIAAFYKESHEHSKQKIDWLGAVTLVGAIICLMFALELGGKEYAWDSAIILGLFAGFAVLA
VLFLFIETRVAEPIISFGMFKNRLYATSNATAIFSGAAFITASVYIPIFIQGVLGGSATNSGMVLLPMMLGSVVTATGGG
FLMNKVSYRTIMIPTTAILVAGTLLLTTLNADSPRWLVTIYMVIVGLGIGASFSVLSNAAIHLFDARQRGSASSTLNFLR
SLGMTLGITIFGIIQSHAFTRKLTESFAGQAQAPQGLDTSDARSLLDPAKRALIPAPVLDKITQALSSSIVHTFAWTVLP
AALALVCAIAMSREKLDPNQQHGPASH
>lcl|NZ_CP026520.1_prot_WP_042234305.1_5613 [locus_tag=PC41400_RS28325] [protein=siderophore ABC transporter substrate-binding protein] [protein_id=WP_042234305.1] [location=complement(6267030..6267995)] [gbkey=CDS]
MSKKLSLLLMTLVLAVMAAACGNKEAATTSSKEGTAPTAAASKELTIKHQLGETKVKTNPAKVVVFDMGVLDSLDKLGVE
VAGVPYDGLPNYLKKYKEAKYTNVGGLKEPSFEVINKLKPDLIIISGRQQDAYKELSSIAPTIFMGVDAKRYMESFKENA
KQLGQIFGKEAQVDQELAKVDEAIKGIKDKVGAGSKNGLITLVNGGKISAYGPGSRFGIIHDVFGVPAVDKTIEVSTHGQ
SISNEFIVEKNPDYLFVIDRDSAVGTESAGAKKVIENELVKTTKAFKEGNIIYLNPDYWYLSGGGLASVAEMVKEVSAGI
K
>lcl|NZ_CP026520.1_prot_WP_042234308.1_5614 [locus_tag=PC41400_RS28330] [protein=ABC transporter ATP-binding protein] [protein_id=WP_042234308.1] [location=complement(6268112..6268870)] [gbkey=CDS]
MVEVKGVSKLYGSKHVVENVSVQIGKGKITSFIGPNGAGKSTLLSMVSRLIDKDQGEVLIEGKEIGQWQSNDLAKKISIL
KQSNHINIRLTIRDLVGFGRFPYSQGRLTAEDKKYIDEAIQYMDLGDIQDKYLDELSGGQRQRAYLAMVLAQDTEYILLD
EPLNNLDMKHSVQIMKVLRRLVNELGKTIVIVIHDINFASCHSDYIVALKNGRVVKEGTTDEIIDSAVLKEIYDMDIEIH
NLNGNRICTYFT
>lcl|NZ_CP026520.1_prot_WP_042234310.1_5615 [locus_tag=PC41400_RS28335] [protein=iron chelate uptake ABC transporter family permease subunit] [protein_id=WP_042234310.1] [location=complement(6268864..6269817)] [gbkey=CDS]
MNSQAKVKISILAVLALVLIAVFLLIDASGNWDYVLPRRGKKILAIVITGGAIAFSTMVFQTITNNRILTPSIIGLDSLY
MLFQTAIVFLVGSASFLMNKNVNFLISVGLMVAFTGVLYKLLFKKQGQNLYFLLLVGLIFGTFFQSISSFMEVLIDPNEF
LIVQDKMFASFNNVNTDLLVISVVMLALVGVYFARFTKYLDVLALGREHAVNLGIDYDYVVKRLLVVVAILISIATALVG
PITFLGLLVANVSHEFLKTFRHKVLITGSILISIIALVGGQLIVERVFTFSTTLSVIINFIGGVYFIYLLLKEKKSW
>lcl|NZ_CP026520.1_prot_WP_009673533.1_5616 [locus_tag=PC41400_RS28340] [protein=ABC transporter permease] [protein_id=WP_009673533.1] [location=complement(6269814..6270764)] [gbkey=CDS]
MKKRYLVIALIVLSVASIFIGVKDISPLDLFNLSDDKVHILLVSRLPRLISILIAGMSMSVIGLIMQQLSRNKFVSPTTA
GTMDSARFGILVSMMLFTSAGPLEKMLIAFVFALLGTFIFMKILDRVKYKDTIFIPLVGLMFGNIVSSITTFFAYKNNLI
QNMSSWLQGDFSMVMKGRYEIMYISIPVLVIAYLYANRFTIAGMGEEFSTNLGLKYKQVVNIGLVIVALVSSVVILTVGT
LPFLGLIIPNIVTLYMGDNLKRALPHTALLGAVFVLFCDILGRIIIFPYEISIGLTVGVIGSGIFLYMLLARRKAL
>lcl|NZ_CP026520.1_prot_WP_042234313.1_5617 [locus_tag=PC41400_RS28345] [protein=DinB family protein] [protein_id=WP_042234313.1] [location=6271488..6271985] [gbkey=CDS]
MVHVKDILSDQLLANANDPSWYLPFSAAVENLSEEAAFWKPSEDSYSIAEIVQHLLYWNETWQTRYQKSDVNTVPSIGNN
NKSFIIPKDKKFTDLKDQLLDVLLKWQDLLTEEKVESDVIGFPVAAKWWELLGNLSTHNAYHIGQICYIRKLQKSWNVDE
KQEII
>lcl|NZ_CP026520.1_prot_WP_042234314.1_5618 [locus_tag=PC41400_RS28350] [protein=cupin domain-containing protein] [protein_id=WP_042234314.1] [location=6272187..6272747] [gbkey=CDS]
MYYTRNTFPYPYYVHAPVYNYARQPVYRGIPHQLKNTDRFGHRHPSNANGKTELKDYGKTPFVVNINEAAKQNNTYRTAI
WTGDHLQVTVMSLNPGEDIGLEVHPTTDQFLRVEQGQGLVQMGDTKDHLNVQKRVYDEYAIMVPAGKWHNVTNTGRTPLK
LYSIYAPPQHPFSTIHRTKADAMAAE
>lcl|NZ_CP026520.1_prot_WP_042234315.1_5619 [locus_tag=PC41400_RS28355] [protein=alpha-mannosidase] [protein_id=WP_042234315.1] [location=complement(6272881..6275604)] [gbkey=CDS]
MSKKRTAHIISHTHWDREWYLPYEKHHVRLVKLVDTLLDKLDAGPDYKSFYLDGQTIILEDYLQVRPEQKERLEKYIREG
RIFIGPWYILQDAFLTSPEANVRNMQIGHQDAKKYGQISKIGYFPDTFGIYGQAPQILQQAGIRNAFFGRGVKPTGFNNT
VSDSGYESSFSELIWEGTDGSKVLGVLFANWYSNGNEVPAGEEEAKEYWARKLADAEKYASTGELLFMNGCDHQPIQLDL
PEAIETARKLYPDTEFVHSNFAAYLEALGDSVERELSSVKGELRSQRTDGWSTLVNTASARVYLKQMNQEGQALLEKVAE
PLASFAHLLGEQYPHHLFTYAWKTLMQNHPHDSICGCSVDEVHREMVTRFEKSRHVAESIVDDSKRAIAEAIDTSAFAGF
GEDALPVVVFNTTGWERTGTVSIELDAARLYFREGYSLEETASRMKNVDLSGRVLVDAEGKVIPCTVEDLGLLFGYDLPD
DKFRQPYSCRKVKLTFEARNVPALGLQTYAWVRSDKAPSAPASLFAGERVLENESLKVEIAQDGSFALTDKANGRTYRDL
GVYENTGDIGNEYMYKQPEGETALTTKGLPAQVRVLEDTAFRASVEIVHEWEIPVSADETLDREQREIVYFPHRKAQRST
ETVPMVIRTVVSLDRGGKGVELSVSFNNQAKDHRIRALFPTDLQASSHAADSVFEIARRDNEPAAEWENPSNTQHQQAFV
DVSEAQAGLTVANLGLSEYEVLRDGRNTIAVTLLRAVGELGDWGLFPTPEAQCLGEHTVRLMLIPHSGDGISSGAYAQAY
QFQIPWTLCQTGVHTGAVQPVYAPVQWKGRELAFSSLKMNEESGDILLRWFNMSSEQAELPVRTSLPHSGFYKSSIMEEA
GEAIPAESGAIRAQVAPHEIVTIGLRR
>lcl|NZ_CP026520.1_prot_WP_042234316.1_5620 [locus_tag=PC41400_RS28360] [protein=beta-galactosidase] [protein_id=WP_042234316.1] [location=complement(6275794..6279093)] [gbkey=CDS]
MSVNQPVVVFYDPSFPISTADQTAVKTGRQAVPVLKEEVRQPDSAAGLTDAAGAALQTDTLAGIGNFVSADGLAEALNEA
AGGCFVNLHAPYFPKAAWPAILGFLSRGGSLISVGGAPFKHPVRREGGEWHVEAEQTAYHQELHIHEALRVDASSAAELS
ASDTVPLLAGQESLFSVADTWNLVPHVTKTSDLPHQMGSAGPMSTQIYPLLKGTTAEGREVSAPVVLWENSRGKFEGSRW
LFVNQPLRAQFWEQGGAAALKQWTEFCAAGVTELWLKPNYASYEAGERPQLTLQTQRMTRVNGAGRVKKAENWTFSITVE
HESTSDRWTCELQQEVTGEFGLVRIPVPLDIRSGLYRVECRAESASGEIRNLRQGFWGLDAGLLAEGRPIGRDRDYFIQD
GRPLPVVGMTYMTSDVARKFLFLPNAAVWDRDMAQMARAGINWIRTGIWTAYRNIMQVDGHVSEDVLRAIDAFLMTAKKH
GLQVTFTFFSFTPETWEGVNPYLDPQSVEAQKRFIRSIVSRHRNTKNVDWDLINEPSMFEPEKIFSDGPRSCRDPFEKAA
FIEWLRERHGSVEALQERWNMSPEQLPDFAAAVVPEAEEINFDVQDMHRGKKGTRWLDYCLFSMDMHNRWAKELYETIKE
QCPDHLVTVGQDEALGAQRPSPFFYEEAVDYTTVHSWWLNDQLLWDGIFAKTADKPNLVQETGIMYVETPDGRAKRTETE
LRNLLERKYAYAFATGGAGAVHWIWNTNFYMDNANESHIGALRADGTEKPEADVSYDFGTFMAGIRDLFRERKLEDIVAV
FPYSNDFSNRKLAFDATTRLTRVLAYQLNQPFRAASEYHLDALEQGPVKLILLPSAHNVDDQAMAELVEFVKRTGAVLLA
TGPLGIDAYWRETARLDDVLGSGKLANVRREEMLSIGGSLQPVSFGQRRIAETVKEIRLNDAGKEGRIGAGAATGAATGV
TGQEASSPQGDAVVDIPLGKGRLIWSPLPVELNERTEAVAELYRYALEAAGCESGLEWRRGGNLPGVYGRKISFKEGSLF
VFVSEFAWDAPVEVKDTETGTVYSFLLEKERSVLFATDSAGKLLAVYRPDEVQIGVGSE
>lcl|NZ_CP026520.1_prot_WP_042234317.1_5621 [locus_tag=PC41400_RS28365] [protein=glycoside hydrolase family 125 protein] [protein_id=WP_042234317.1] [location=complement(6279090..6280409)] [gbkey=CDS]
MEQFRLPKIPMPKLELPQAIQEVLKEAEEKLAHRPKLLQLFKNCFPNTLETTTKLLDDGTTFVITGDIPASWLRDSVEQV
IQYVPFAKEDKDLQRIIGGLIKRHIAYIHIDPYANAFNESANDWHWNTTDITDMSPWVWERKFELDSLCFSMRLAYAYWK
ETELTDIFDAGFKSAMRKIVELFKTEQCHFEQSPYRFTRDNGIPTDSLRNSGLGMPVNYTGMVWSGFRSSDDACDFHYNI
PANMFAVVALRHMEEFAEWVFRDMEFLKELKDLEADIDHGIQLYGIYRHPEFGPIYAYETDGFGNYCLMDDAGTPGLMSI
PYLGYVNADDPIYQNTRRFALSKENPFYYEGTAAKGIGSPHTPPGYIWHMALSMQGLTATTAEEKLEMIAMLEATDADTG
FMHEGFHADDPNTFTRTWFAWSNSLFSQLVYRAMKDGLL
>lcl|NZ_CP026520.1_prot_WP_009673528.1_5622 [locus_tag=PC41400_RS28370] [protein=carbohydrate ABC transporter permease] [protein_id=WP_009673528.1] [location=complement(6280605..6281525)] [gbkey=CDS]
MQAQTRKKSRDFHQLPTVWNVVFNLIAALFAFACVFPFLFVTIISFTDETTLAANGYQLIPEKWSLEAYRYLFKAGDQLL
RSYGVTILVTIVGTVISVIMTALFAYAISRKTFKYRNFFGFFAFFTMLFNGGMVPTYIVVTQLLGMKDSLWALILPLAGN
AFYIMIMRTFFSTSVPDAIVESGKIDGAGEFQIFYKLVMPLALPGMATIGLFSTLGYWNDWFNALLYIDDPKLVPLQSML
MRIENSMQFLLQNTNNPSLGMGVLQSLPQDTSRMAMVVLATGPIILAYPFFQRYFIQGLTIGAVKE
>lcl|NZ_CP026520.1_prot_WP_231392072.1_5623 [locus_tag=PC41400_RS28375] [protein=ABC transporter permease subunit] [protein_id=WP_231392072.1] [location=complement(6281556..6282485)] [gbkey=CDS]
MTNFFKDINKNKALLLMVLPGTIWFLFFSYLPMAGMVVAFKQYRYSRDGFWASIVESKWVGLQNFKFLFSTNDAYIITRN
TLLYNSVFIIVGLVLAVLLAIVLSEIVNKRLAKIYQTGMFLPYFLSWVIVGYFVFSFLSFDKGLVNKVGAMFGVDPTQWY
NNPTYWPLIIILVFLWKSVGYNSVVYLAAIMGIDKSLYEAAMIDGASKWQQIIHITIPMLKPLMTILTLLAIGRIFYADF
GLFYQVPRDSGTLYSVTNVIDTYVYRGLKTTGEIGMSTAAGLYQSLVGFILVITSNYIVRKFNKENALF
>lcl|NZ_CP026520.1_prot_WP_042234321.1_5624 [locus_tag=PC41400_RS28380] [protein=ABC transporter substrate-binding protein] [protein_id=WP_042234321.1] [location=complement(6282599..6284068)] [gbkey=CDS]
MKRSKKSLAVVLAMLLTFTLVLSACGGKDGKQAEGTSDKPVELIWYTIGTPQKDTDRVMEEVSKYTKEKIGVTVKMRQID
FGDYSQKMQVMAASGEPIDIMFTSSWAFDYVQNARKGAFLAIDDLLNKQGKGIKDALDPAFLEGSKVDGHNYGIPANKEL
PAQEVWRFNKQLLDKYNLDISNVNSLESLEPLLKTIKEKEPNVFPLAADKNFGLYVPYDYVIEKMPMAVKLDSTDLKVVN
VFDTPEMKQALQTMNKYYKAGYIPTEAATLNSLSDVQATGKWFADKATNQPFADNQWSASYGYPVVSTPAGPSTIFNWSV
MGSMQAISANSKYPEKAMEFLNLLNTDPKLRNMVDSGIEGVHYKKIGDNVMENLPDAKNYDMPTFSLGSVIITYLNKEDP
ANKWDEFKKFNDAGKPAPLLGFNFDTSKVTTEIAAVQNVKEAFWTSLMTGTVDPNQYLPQANEKMKAAGLDKIMAEAQRQ
IDEWKASKK
>lcl|NZ_CP026520.1_prot_WP_042234323.1_5625 [locus_tag=PC41400_RS28385] [protein=response regulator transcription factor] [protein_id=WP_042234323.1] [location=complement(6284315..6285835)] [gbkey=CDS]
MYRVFLVDDEPFIIEGLYDIIDWSSFGLEIVGQAENGREALEALKKTPVDLLITDISMPVMNGLDLIRGAREFHPDLKVV
VLSGFNDFVYLKECIRLGIENYLLKPINLDELKATLDNTVEKLNASKSERLFNEYGVQILKDNMMHRWLTEQIASGEYRE
RADLLGIEMDKPFMAAAVMRLEENFVQVFELISRQMKYNESIIPFRDVDGDLVILFVMDDPEEGKREMIGTIEELLDRLT
TYQPIRTSIGSVQPLPDQAPLSYRHAKKAQEYFLLYPEHGLIDYSDLPSGKDAGKATFPIDWSEYTKLIMAKDREKLLLR
IEEDFEEMQTLEGVTPGDIQDIAMEVIILFKMALKEIKHVEETELYKEGFEKVRQAGTLRELIKAVQDVAGLTVDSLIRD
IKSPVVQQVLNYIDESYAEELSLKALGALYNIHPVYLGQLFHRETGETFTEYINKYRIEKAKEQLKTTHLKVHEIARNVG
YWETGYFYKQFKKYVGISPTDFKGLL
>lcl|NZ_CP026520.1_prot_WP_042234325.1_5626 [locus_tag=PC41400_RS28390] [protein=sensor histidine kinase] [protein_id=WP_042234325.1] [location=complement(6285828..6287648)] [gbkey=CDS]
MGVRRIYRNYLKNNLFLKMILMFSLITILTIITLSYLMFSFMSQSIVQRELANQKKAMEGVSSYVNGKFEWMQSLLTDIY
QNDALSANMSYFLQHPYQEYAEHRLDQFYKDTNFSTDVLKYLQNKLDVDPDIQNLMLYSAEEQYLYTYNSNRLSKLISTN
ASRSYIPDIMAQETRTIALPNIWMRKTLNQWDPRLYSIRVPVNDKYSLKNIGQLLLYFNSDGIVRALESYKPDLKGTIFV
MTPDGDVMFDSSGQYYGKKFPYTDRINSLYDTGTLQNGMYLTKLTQSRGGFIVMGAAPKEEIAASYRGLRNTILLVSAIA
ILFAILIPSLFIMNFAKRTNRIIQFTRRVKRGDLKARIRDTREDELGQISKSFNDMLDELNNYIDRVYKAEIKQKHTELT
ALQARINPHFLYNTLEVIRMRALSQGASDVGEMIYSLSVLFKSFVQQKTIYTMKDEVEACRMYLELFRIRYKDRFSYEIV
WERDLAHVQILKMSLQPIIENYVVHGIRSDRKDNRLTITIGEEGGFVRVRVQDNGNGMTEEKLAEITKKLDREETEGESF
GLRSVHERLKLQYGSEYGIELHSEPGQGTTVIVRYPGAEGQEAAHV
>lcl|NZ_CP026520.1_prot_WP_042234328.1_5627 [locus_tag=PC41400_RS28395] [protein=copper amine oxidase N-terminal domain-containing protein] [protein_id=WP_042234328.1] [location=complement(6287891..6289489)] [gbkey=CDS]
MQMKKLLLPALSLSILMPTVAANAAAVVNTPKPTVNTPAADLRASLDHLLSEHFALAVTAMTKAYDGSADAAEAYKALDQ
NALDMTPAIASLYGKEGAAEFERIFRAHNKYTDDLVKATKMNNQEARKKAEEQVQGFVDEFSAFLDKATAGKLPKQAAAQ
AIRLHEDQVQETFDDYIAGDYQGAYAAYREGFKEMFTISKALSTAITTQMPEKFGNTKADTPAADLRSALNNLAAEHLSL
STLQMQKQYDGSKDSDALIKAESGNTADFKAAISSIYGAEGGMAFEKIWVTNHVNAQSDYVTAVKNKDQAAKAAVEKRID
GFTTEFANFLGTATAGNLPAAAAQDVLRQHEGQVQKVLNEYAAGNYEASYKTDREGFKLMFGVGQALGNAIVTQYGEKFQ
EKTDPSMTTVWMQIGSHDLNINGTVTKMDTTPFIWNDMSYIPLRYLAEGIGAEVTWDNEAQSVWVKAGNDKMQFWIGKDF
MELNGMKKNVGSKVFVNTDGRTVLPLRFITELLGWDVKWGQQDRSITLTKSM
>lcl|NZ_CP026520.1_prot_WP_042234329.1_5628 [locus_tag=PC41400_RS28400] [protein=hypothetical protein] [protein_id=WP_042234329.1] [location=6289790..6290386] [gbkey=CDS]
MTKKTITGIAALAAVVISSTVVYAANTTTVSVPTDTARSFTEKYTDAVAPEASSKPVPYKEFSDSRVYSKLESSWTKAVQ
ALHTKASKEANQDTKQPAAFAQEQWNLIAGPDIEAEFESKLSRNEASSFNEVFITTTRGLKPGDLMYALLLKGNKTEAKV
VWQRANGDIHVSHLRQSKDLHGNIVWSLLGSEEIASAE
>lcl|NZ_CP026520.1_prot_WP_042234330.1_5629 [locus_tag=PC41400_RS28405] [protein=DsrE/DsrF/DrsH-like family protein] [protein_id=WP_042234330.1] [location=complement(6290818..6291300)] [gbkey=CDS]
MVEQKEGTTIVLFSGEMDKAIAAFIIANGAAAYDHEVTIFFTFWGLNALRKDEPVKLKKGWLEKMFGRMMPRGPKKLGLS
KMNYMGMGPKMIQHVLKKHNALTLPQLIELAQEQGIKLVACTMTMDLLGLQKEELLDGIEYAGVAAYLGDASQAKVNLFI
>lcl|NZ_CP026520.1_prot_WP_042234333.1_5630 [locus_tag=PC41400_RS28410] [protein=sulfite exporter TauE/SafE family protein] [protein_id=WP_042234333.1] [location=complement(6291380..6292171)] [gbkey=CDS]
MDVLLAVIMLALGLAGSFFSGLLGIGGAIINYPLLLFVPPLFGEAAFTAQEVSAISMFQVFFASLSGVLFRRRGKNGPSL
VHKELVLYMGGGVLAGSLIGGLASKYMPGDTINVIYGVLAIMAVVLMLIPVKGKRQEEGSGLAFNKPLAVLSALAVGLVS
GIVGAGGAFILIPIMLTLLKIPTRTTIASSLAIVFISAVGGVVGKIGAGHIPLLPVIFTVIGSLLGAPLGSRLSARINVK
YLRYALIVLIAATAFKIWLPILL
>lcl|NZ_CP026520.1_prot_WP_164977160.1_5631 [locus_tag=PC41400_RS30030] [protein=hypothetical protein] [protein_id=WP_164977160.1] [location=6292322..6292492] [gbkey=CDS]
MIQQLKRLFVSVCPHCGQPLQTDEATAHCVIKVCPGGHYTEESYGHLGVRVVYTDN
>lcl|NZ_CP026520.1_prot_WP_009672989.1_5632 [locus_tag=PC41400_RS28415] [protein=sulfurtransferase TusA family protein] [protein_id=WP_009672989.1] [location=complement(6292591..6292818)] [gbkey=CDS]
MEVNVTVDTKGMACPMPIVKAKKALDSLESGGIMQVLSTDKGSLNDFQAWVKQTGHELIKHEEDNGVYTFYVRKK
>lcl|NZ_CP026520.1_prot_WP_042234335.1_5633 [locus_tag=PC41400_RS28420] [protein=MBL fold metallo-hydrolase] [protein_id=WP_042234335.1] [location=complement(6292876..6294036)] [gbkey=CDS]
MNTSNREEGRLRGLTAAALAEKVLSNKPLFILDVRNESDFADWKIEGNRVDMLNVPYFELLDGVEAVTDKIPTDREVLVV
CAKEGSSKFVGEQLVEAGFSDVSYLVGGMKSWSEHLEPVKVGDLKDGGAVYQFVRIGKGCLSYMVISGGEAAVIDAVRMT
GVFTAFAGEKQAVIKHVLDTHLHADHISGGRKLAAETGAAYWLPPKDAEEVVFGYSELEEGADMTVGSTRIAIQPMYSPG
HTIGSTSFVVDDRYLLTGDILFVASIGRPDLAGKAEDWAGDLRATLYDRYKKLSEELIVLPAHFGRFAELGEGGRVSARL
GDLFANNPGLNIGDEGEFRKTVSENLPPQPNAYQEIRQTNMGKITPTEEEQREMEIGPNRCAVHDQ
>lcl|NZ_CP026520.1_prot_WP_042234336.1_5634 [locus_tag=PC41400_RS28425] [protein=sulfurtransferase TusA family protein] [protein_id=WP_042234336.1] [location=complement(6294146..6294724)] [gbkey=CDS]
MNIQTHQILDAKGLACPMPIVRTKKALEQMASGQVLEIQATDKGSLADIRSWAKTTGHQYLGSTEEGEVLRHFIRKSDPS
EVKEERQFPHAATNDELERCLREKPQTNVLDVREPAEYVFGHIPGAVSIPLGELEQRIGELNAHEELYIVCRTGSRSDMA
AQLLERQGFKRIRNVLPGMSAWTGPLEKSAGI
>lcl|NZ_CP026520.1_prot_WP_042234337.1_5635 [locus_tag=PC41400_RS28430] [protein=ring-cleaving dioxygenase] [protein_id=WP_042234337.1] [location=complement(6295147..6296121)] [gbkey=CDS]
MELKGIHHVSAITAQAPENFEFYTKVLGLRLVKKTVNQDDISVYHLFYGDEKGNPGTELTFFEIPMAARNREGVSSISAA
SLRVAGDEALSWWKKRFTEFGVDHDEIQDRAGRATLAFRDFEGQRLILVSDENNMGVQGGIPWDRSPVPAEYGILGLGPV
KLTVQDPEPTVIVLTELLGFREKGRYAADVPGQPDIVVYETGEGGSGAEVHIEERNDLSREQLGRGGVHHVAFRVDNEEE
LRAWINRVRQVRFPNSGFVDRFYFRSLYFREPNGILFELATDGPGFDTDEDIRHLGESLALPPFLESKRPQIEAGLKPLN
TKAD
>lcl|NZ_CP026520.1_prot_WP_042234340.1_5636 [locus_tag=PC41400_RS28435] [protein=MarR family transcriptional regulator] [protein_id=WP_042234340.1] [location=complement(6296147..6296596)] [gbkey=CDS]
MEKNKKESFLSKQEEKLGVLIWFRLSRVYNQSIRQSNQHLKAWNLSAAQFDVLVQVGSHERLTQQELADKLLVTKGNITQ
LLGKMEELGWIRREQEWKTKYISLTAEGQALFDEVVPQQEQFQASQFDGLDREEKKQLLELLKKVQKQS
>lcl|NZ_CP026520.1_prot_WP_042234342.1_5637 [locus_tag=PC41400_RS28440] [protein=hypothetical protein] [protein_id=WP_042234342.1] [location=complement(6296850..6297947)] [gbkey=CDS]
MKAAEKARINPSPFPRTGTAVAKSMPEQKGSKMSIIGKILFWAFAVMVIFLTMGVILLVFDKRKLTQKANELEGELFHME
AKILPLTELYKGRATEKNLSPVKGELQAAWEELEDLRKRIATIFTQKDSSNRIKVILEDMAALEKSCGSLKKKIEKLIAM
EHENESILLQLQTKGKQLMKRLEALSGSHQLAEVQKGLAAQMALIEAIDDKQLTDLINVHDTLMPIRAQMDEIENLLDSL
PLNYELGQVKQGEEIEDQVEVVRAQMEQAMRASGLKKEEKPQDNKNDAQTSSVHVTRQASAESIDLDEVKRLMDQGLVFD
AIHRMAASLDPTVKTRHDVQVKVVVNSPHPEQQESKKQKRGKKSG
>lcl|NZ_CP026520.1_prot_WP_042234344.1_5638 [locus_tag=PC41400_RS28445] [protein=hypothetical protein] [protein_id=WP_042234344.1] [location=complement(6297969..6299747)] [gbkey=CDS]
MKKIWVRNLLAFSLLTGIAHTSFMTQEAAAESSRTSAVQGYTVQSGTIREAGGMTAAAPLLADGMFVAVSDKDENVNHIG
LKELGIFAGIIFVIYQSFRIFRQRKRKEKLLGEAQAYSTDLYRDSERLGLLADLYKGPSAEKRIRPVEAELKDVSSILDE
MTAMISGIKISFIGNVKTMEETMGECRVLLRQYQGVYGSLKASIDKLCELEKQNDKVVSELQAKIESLLARAKDRSAAYR
VDDLEEELLSLQKRIGIIDREQMSDFIDVQEHLAPEREQLAEIERLLAELPELHNRQQKFKGQIESARDDVNRSIREHGL
DSSDLTSFASLDRAEALSAAMLEALNAGRIREAARISSDVTGLIKHSVIAAGELAELRESLTKTLRLLKSGLRELDYPES
AFKEELKRVKEYFAESVWSFMRGEFDQFTALVREITPKLPEVENMLAEGRFNEGGKKLIALMEKYELAEKKYVRLQVYEA
ANKELNGLRERVHACWMDFQETVSFISSEGLEEALHSRDLGDLKYKIHDRKEMIDRRMSSPPIQLDRLRRDAEAMSADIQ
AYRARVEQIADQQEEYEDEEESKSVVINVNFK
>lcl|NZ_CP026520.1_prot_WP_042234346.1_5639 [locus_tag=PC41400_RS28450] [protein=virginiamycin B lyase] [protein_id=WP_042234346.1] [location=complement(6300032..6300925)] [gbkey=CDS]
MSYTFEEFVLPDKDSGPYGLTVGKDGAIWFTEQKGNRIGRMAPDGTITEYPVTAEEAGLSVIASAADGALWFSGYKAGIV
GRIDLSGEITGFPLPAADSGPFGLAEGPDGAVWFTEMTGNKIGRITADGTITEYGLPLPGAFPSFIARGPDGAMWFTENQ
GNRIGRITPAGEITGYPLPMGQAGPVGITAGPDGALWFAEINGNRIGRISTTGGITEYSLPEPGARPHAIAAGTQGDLWF
TEWGSNKIGRITTTGEISEYAIPREKAEPHGLAVGPDGAVWFAQESGQIGRLTVRSF
>lcl|NZ_CP026520.1_prot_WP_042234349.1_5640 [locus_tag=PC41400_RS28455] [protein=MFS transporter] [protein_id=WP_042234349.1] [location=complement(6301138..6302526)] [gbkey=CDS]
MSTAVQKTTGDKIMRVLAFTLVISVMNASIFNIVLPQISAEFHLNSAQVSWVASIYMLVYAIGSVIYGKLADKYPLKRLI
TFGLTVFALGSVIGLSATAYWMVILARGFQAVGSSVIPAAAMIIPIRFFPPETRGRALGISAIGMALGNALGPVLSGLIM
SVADWRWLFCLSLLTLVTLPFYRKLLDDKPGTAGKIDAWGGALLAGTVSLLLLSITNGGWITVISGIVLLGLFILRILKA
EEPFIQPRLFLNKHYVTGLVVSLIIVGLGFSLSFLIPQLMSHVNHIEPGMIGFMMVPGAAASAMMGLRGGRLADEKGNSF
LVYLASALLFISFVLLSSLSGTLPALVSALLIFGNVGMTFMQIALSNTISRSLARDQVGVGMGLMAMMNFIGIASSSAIY
SKLLDNGATVTWNPFNLYEGASVYSNIYLGLAVLVLAAAGIYFYRFGRAPEAAGTAPRKAEA
>lcl|NZ_CP026520.1_prot_WP_042234350.1_5641 [locus_tag=PC41400_RS28460] [protein=TetR/AcrR family transcriptional regulator] [protein_id=WP_042234350.1] [location=complement(6302634..6303230)] [gbkey=CDS]
MSGSSSSSSSDKLLLAAIDLIADRGYKCVTTKEIAAAAGLSEMTLFRHFGSKQNLLESAFDRFHYAGEMQKLFAEKLVWE
LETDLLLISRTYHDIMNRNRKMLQISLKESGSLPGFREKTQRHPRQLKEFLTDYFVKMAERGKMISTNPELQAVSFMWMN
YGAFMNNRDGWGEGTFAEIPLDDFIRESVRVFTKGLAP
>lcl|NZ_CP026520.1_prot_WP_042234352.1_5642 [locus_tag=PC41400_RS28465] [protein=pyrimidine-nucleoside phosphorylase] [protein_id=WP_042234352.1] [location=complement(6303536..6304837)] [gbkey=CDS]
MRMVDLIAKKRDGKELTTEEINFIIEGYTKGDIPDYQVSSLAMAIYFKDMTARERADLTMAIVNSGETIDLSAIEGIKVD
KHSTGGVGDTTTLVLAPLVAALDIPVAKMSGRGLGHTGGTIDKLEAIEGFHVEISKDEFVNLVNEHKIAVIGQTGNLTPA
DKKLYALRDVTATVDSIALIASSIMSKKIAAGSDAIVLDVKTGAGAFMKTPEDAKELAHAMVSIGNNVGRKTMAVISDMS
QPLGAAIGNALEVQEAIDTLRGQGPKDLEDLCLALGRQMVFLANKASSLEEAEEMLKEVIRNGKALEKFKEFIANQGGDA
SVVDDPEKLPKAKFLIEVPAREDGVVAEIVADEIGTAAMLLGAGRATKESEIDLAVGLMLNKKIGESVKAGESLVTIHAN
RDNVEDVIRMIYDNIRIADHAEAPVLIHDIVTE
>lcl|NZ_CP026520.1_prot_WP_042234355.1_5643 [gene=deoD] [locus_tag=PC41400_RS28470] [protein=purine-nucleoside phosphorylase] [protein_id=WP_042234355.1] [location=complement(6304965..6305672)] [gbkey=CDS]
MSVHIGAKPGDIAETILLPGDPLRAKYIAETYLSDVICYNEVRGMLGFTGTYQGKRVSVQGSGMGIPSISIYANELISQY
GVKNLFRVGTCGAMQEHVRVRDVILAQASCTDSSANRHVFGGFDYSPIASFPLLKAAYERGVEKGLKLHVGNIFSSDIFY
RDDKTVVQKLMDYGVLGVEMETTALYTLAAKFGVNALTILTVSDHLLTGEETSSEERQTTFNEMMEVALDTAISL
>lcl|NZ_CP026520.1_prot_WP_042234358.1_5644 [gene=deoB] [locus_tag=PC41400_RS28475] [protein=phosphopentomutase] [protein_id=WP_042234358.1] [location=complement(6305726..6306907)] [gbkey=CDS]
MTTFKRIHLIVLDSVGIGEAPDAAKFDDFDVDTLGHIARERGGLNMPNMAKLGLSNIRPIEGVPAADKPLAYYTKMQEAS
NGKDTMTGHWEIMGLNIETPFRVFPDGFPDELIGRIEEKTGRKVIGNKPASGTEIIDELGEEHVKTGALIIYTSADSVLQ
IAAHEDVVPLKELYEICEFCREITLEDPYMLGRIIARPFVGEVGSFKRTANRHDYALKPFGRTTMNELKDRGFDVIALGK
ISDIYDGEGVTKAVRTVSNMDGMDKLVDTLGESFTGLSFLNLVDFDALFGHRRDPQGYGQALEDYDARLPEVFAKLTDED
LLILTADHGNDPTYRGTDHTREYVPLLVYSPRFADGGKELPLRRTFADIGATVADNFGVTLPAHGTSFLADLK
>lcl|NZ_CP026520.1_prot_WP_042234361.1_5645 [locus_tag=PC41400_RS28480] [protein=NupC/NupG family nucleoside CNT transporter] [protein_id=WP_042234361.1] [location=complement(6306934..6308112)] [gbkey=CDS]
MKYLVAILGVLVVFGLTYAASNNRHKIRYRPLFVMIALQVVLAFALLNTSVGEFLIRGFASIFESLLAYASEGINFVFGG
IANDKTSPFFLNVLLPIVFISALIGILQYTKILPFIVKYIGLVLSKINGMGKLESYNAVASAILGQSEVFISVKKQIGLL
PKHRLYTLCASAMSTVSMSIVGAYMQMIEPKFVVTALVLNLFGGFIISSILNPYVVTKEEDILEVQEEEKQSFFEMLGEY
ILDGFKVAIIVAAMLIGFVALIALINGIFDGIFGISFQSLLGYIFAPFAFVMGVPWSEAVEAGSIMATKMVSNEFVAMLD
LAKHTTMSPRAVGIVSVFLVSFANFSSIGIIAGAVKGLHEKQGNVVARFGLKLLYGATLVSVLSATIAGLFL
>lcl|NZ_CP026520.1_prot_WP_042234363.1_5646 [locus_tag=PC41400_RS28485] [protein=cytidine deaminase] [protein_id=WP_042234363.1] [location=complement(6308235..6308636)] [gbkey=CDS]
MKDQLIQEAIEARKQAYVPYSGFQVGAAVLTGGKVYRGCNVENASYGLTNCAERTAIFKAVSEGEGGHKLEAIAIVADTE
GPVSPCGACRQVIAEFSDLNTKIYLTNLHGNTEEWTIDRLLPGAFQADDMEKK
>lcl|NZ_CP026520.1_prot_WP_042234364.1_5647 [gene=deoC] [locus_tag=PC41400_RS28490] [protein=deoxyribose-phosphate aldolase] [protein_id=WP_042234364.1] [location=complement(6308664..6309329)] [gbkey=CDS]
MSIAGMIDHTLLRADATKAEITKLTEEAKKYEFASVCVNPTWVAYSAEQLSGSSVKVCTVIGFPLGATTSAVKAFEAKDA
IANGAGEVDMVINIGALKAGDDTLVESDIKAVVDAAAGKALVKVIIETSLLTDEEKVRASELSVKAGADFVKTSTGFSTG
GATPADVALMRKTVGPDVGVKASGGVRSLEDMNAMIEAGATRIGASSGVKIMEGGQSTSSY
>lcl|NZ_CP026520.1_prot_WP_042234366.1_5648 [locus_tag=PC41400_RS28495] [protein=TIGR00730 family Rossman fold protein] [protein_id=WP_042234366.1] [location=complement(6309592..6310173)] [gbkey=CDS]
MKRIAVFCGSSNGASEVYREGAVRLGRELAERGLALVYGGASVGLMGAVADAVLEAGGEVIGVIPKMLENREISHHGLTE
LIVVESMHERKAKMAELADGFMALPGGPGTLEEFVEIYTWGQLGLHRKPFGLVNINHYYDPLVALFDRMNQEQFMQDKYR
SMVLVNEDPAKLLEQFASYEAPEVKTYIKKDQT
>lcl|NZ_CP026520.1_prot_WP_042234367.1_5649 [locus_tag=PC41400_RS28500] [protein=Ada metal-binding domain-containing protein] [protein_id=WP_042234367.1] [location=6310400..6310969] [gbkey=CDS]
MNKDAIFKSVYETILSRDTRHDGQYYVGITSTGIFCRPSCRSRTPKPENVRVYDSIERALEAGFRPCKRCKPENPSPHGP
DAELAETVLGLIRTRYGENLTLHGLASELNISPYHLQRTFKRTTGISPSRQLLLTRMEEAKHRLARSEQPIGEIAAAVGF
RSASHFSAVFHKTAGCTPNEFREQNAPNS
>lcl|NZ_CP026520.1_prot_WP_042234369.1_5650 [locus_tag=PC41400_RS28505] [protein=methylated-DNA--[protein]-cysteine S-methyltransferase] [protein_id=WP_042234369.1] [location=6311059..6311625] [gbkey=CDS]
MNTLSATTEIYWAALVHPLFRNKPLYAAATDRGLCRITWPTESLDTLKSWAAKQIPGAVLIEDEERMAGYLLQLKEYLEG
NRQQFDIPLDMRGTAFQTSVWKALAEIPFGETRSYSDIAASVGSPAAVRAVGTANGANPVPIIVPCHRVIGKNAALTGFR
GGLQVKEELLRLEGFHDYTAKGHARFQF
>lcl|NZ_CP026520.1_prot_WP_042234371.1_5651 [locus_tag=PC41400_RS28510] [protein=DNA-3-methyladenine glycosylase] [protein_id=WP_042234371.1] [location=6311585..6312577] [gbkey=CDS]
MTTRQKDTPVFSFDENTGIAQIETPAAFSYAETLLYLTRSPLECLHSVENGRIYKLLEAGKELLPVEISNPGPGRLHLRF
MQGPPADPDAIREAAAYTAEWLDLARDMTPFYATAQSDPLLGPLASGYYGLRIIGVPHLFEALCWAIAGQQINLTFAYTL
KRRLVENYGKKLVWEGRTYWLFPKPGDLADVTVEELRQLQFTGKKAEYILHVAGLMDSGALSKEALLASGSYEAAERTLL
AIRGIGPWTAHYVLMRCLRNPAAFPIGDAGLHNALKQLLHRTQKPALEEIRQLFAPWRGWEAYAVFYLWRSLLPEGSPAI
QTAGLPDART
>lcl|NZ_CP026520.1_prot_WP_042234373.1_5652 [locus_tag=PC41400_RS28515] [protein=4'-phosphopantetheinyl transferase superfamily protein] [protein_id=WP_042234373.1] [location=complement(6312540..6313220)] [gbkey=CDS]
MDIYAINLRLDEKETVQHTDRLIVHLSEEKRAKAQRFYRTEDRVRAVAADALARWCLCRKLAAANPELRFGTNAYGKPHV
TAPRTGVHYNVTHSGCWVAAGVSEAEVGIDVEQIQPVDLDIAERFFSAAEKADLFSLPPEERLNYFFDLWTLKESYIKAV
GKGLSIPLDSFAIRKRGSGVFELEPQPTHVFKQYELDGPYKLSVCSTLDHFPGEVTFLRLEDLLFE
>lcl|NZ_CP026520.1_prot_WP_042234460.1_5653 [locus_tag=PC41400_RS28520] [protein=NCS2 family permease] [protein_id=WP_042234460.1] [location=complement(6313716..6314999)] [gbkey=CDS]
MRGLNRYFQIEAHETTWKRELMAGLTSFFTSAYIMLVNPLILQDTGMPLSAGVLATIAATIIGCLMMALWANAPIIVIPG
MGLNAFFTYTIVQSLGLSWQQALAVVVVSGVIFLLASMSSFGSKILSAVPVSLKHGITAGIGLMLTFIGLQKGHIIQPDS
VNYVALGSFTSPVTLATLIGLAVTFMLFVRNIKGSLLIGIGFTSVLTLGLAGKGDLADKAVDLKAYGQVLGAADFSVMQI
PFWIAVFSMTMIVLFENMGLLSGMLPDKEKFPKAYRVVAVSTIVSGFFGTSPTIASAESTSGISEGGRTGIPPLVTAVLF
LLSALALPVIGLIPGNAVAPVLIIIGALMMMSVKDIPFNDFSEYLPAFLIVVCIPLTSSIADGLAFGFIAYPLTKVAIGQ
WRKCSPLLYAIAGLFVLNFVASAVFVH
>lcl|NZ_CP026520.1_prot_WP_174446852.1_5654 [locus_tag=PC41400_RS28525] [protein=amidohydrolase family protein] [protein_id=WP_174446852.1] [location=complement(6315053..6316366)] [gbkey=CDS]
MVLNTAYVNGHFFTMDESSRQYKSGMMVVSGDAISYIGDHSVQRLAAADRVVDLGGKWVMPGLVNVHSHIVMTILRGIGD
DMLLKPWLETKIWPMEARFTPEIASVSTRLGILEMLKSGTTTFSDMFNPNGIDLGGIMQEIADTGIRGAFSHTIFSLGTE
RQQRDNIASAERFAKQFRTFADGRLTTMVAPHSPYACTPAALEESARVAAENGLMVHIHVSETDLEIRDIESRYGVRPVE
HLRRLGVLDQPTVMAHGVVLNEEERAILHQYDVRVAHNPISNLKLGSGVADVAALLEAGVKVGIATDSTASNNNLDMFEE
MRTAALLQKGIYKDASKLPAETVLALATRVGAEVIGMGHTGSLEAGKKADFITIDPSDKPHLQPLEQAFSHLLYAASGRD
VCDVYVGGAAVVRDGSCLTIDEEKILAEANRLQAELQ
>lcl|NZ_CP026520.1_prot_WP_042234375.1_5655 [locus_tag=PC41400_RS28530] [protein=purine-nucleoside phosphorylase] [protein_id=WP_042234375.1] [location=complement(6316640..6317473)] [gbkey=CDS]
MTQTTQTNHVSQAAAYIKDKLSYTPQVGLILGSGLGVLADEIENPVVIPYHEIPGFPVSTVEGHAGRLVCGMLQGKPVVA
MQGRFHYYEGYAMNKVVFPVFVMKRLGVKTLLLTNAAGGVDVSLEPGDLMLISDHINLMASNPLIGPNDAELGPRFPDMS
EAYSKRLREAARETAGSLNMSLKEGVYAGLTGPSYETPAEIRMVRALGGDAVGMSTVPEAIAASYAGLEVLGISCISNMA
AGILEQPLSHAEVMETTEMVRANFVALIKGIVGRSDL
>lcl|NZ_CP026520.1_prot_WP_042234378.1_5656 [locus_tag=PC41400_RS28535] [protein=sugar-binding transcriptional regulator] [protein_id=WP_042234378.1] [location=complement(6317550..6318515)] [gbkey=CDS]
MEDEKLRKIIEVAKLYYQLDYNQNQIAQMLGLSRPTVSRLLQQAKTEGIVKIHIADPSEDCEALAGTLREKFHLKKAVVA
FVPTYDDEIVKKFIGEAAAHYLMTTVKDKDIIGASWGTTMYQVAANLQQKGLTDSRVVQLNGGVSHADITISPSEIIYLF
SRAFHVTPYFLYLPAIVDHSLVKQAIQSDRHIRRVLDLGKLANIAIFTVGLPDAESVLIQSNYMTEEDLEVIQSRAVGDI
CSRYFDDKGQICHTSLNDRTIGIELEDLKEKEQAILVAGGMRKVNAIYGALRGGYANTLITDQMTAKALIAKHDGGQTAT
A
>lcl|NZ_CP026520.1_prot_WP_129112551.1_5657 [locus_tag=PC41400_RS28540] [protein=hypothetical protein] [protein_id=WP_129112551.1] [location=6318640..6318885] [gbkey=CDS]
MSVNSISTSLHYTIFESVYGGDRSQIDKLCSLSLYMLPQNAMMKDKKVFDSGFTLYDVVEKALIKLGTVNPKVEGRIVEV
K
>lcl|NZ_CP026520.1_prot_WP_042234381.1_5658 [locus_tag=PC41400_RS28545] [protein=VOC family protein] [protein_id=WP_042234381.1] [location=complement(6318980..6319384)] [gbkey=CDS]
MSAKVIHHICIQTDCYEQSLVFYRDALGFEVVQETPGFHTRAYNSWLRLGEFYIELQTGKAGEELQPVNTSSRGIVHFCL
WVEDLEEEVRRLKESGTEFRLKNGEEIYTVENGKLCKLTAPEGTIIELRDTKGI
>lcl|NZ_CP026520.1_prot_WP_042234383.1_5659 [locus_tag=PC41400_RS28550] [protein=putative quinol monooxygenase] [protein_id=WP_042234383.1] [location=complement(6319552..6319848)] [gbkey=CDS]
MIIIHAAMRLQPGKDDAFLKEILPLVEASRGESGNVSYDLYKHAEKEHAYTMVEVWKDQQAVTSHNSSAHFTAFTGKAGE
FLAAPLDVKVYAAEAVKV
>lcl|NZ_CP026520.1_prot_WP_042234384.1_5660 [gene=hpaB] [locus_tag=PC41400_RS28555] [protein=4-hydroxyphenylacetate 3-monooxygenase, oxygenase component] [protein_id=WP_042234384.1] [location=complement(6320042..6321514)] [gbkey=CDS]
MPVKNGSDYIERINRHLPDVWIAGERVRGLISEHQAFKGLMATQASMYDMQQAEEWQAKMTYASPSTGDPVGLSFLQPKT
KKDLALRRQMMQAWASAHHGFLGRAPDYMNTAIMAFAAAAGLIEKETPEYADNLRRYYEYCRENDVTLSHVFIQPKAARM
PAFLRTFEEDAAARVVEKNKDGIVVRGAFLLDTQGATSDEILVFPAPFPYGGGEDSPYAFAFAVPSNLAGVSFICRESFV
GGESAYNYPLSSRFEEMDTLVLFEDALVPWDRVFICGNERMAWQLVDESRFHTHAGMQIMCKNIAKTEFILGTLERMVEV
GGLDSYGHVTEKVTEVIVALETLKAFQIAAEKGASRDKWGSMLPDPKPLLAANSYFPKVYPRFLEIIQLIGASGLIMIPD
EADFGTEIGGQLNLYLKGIDLSAEENVKLSRLAWELSVSAFGGRQAVYERFFFGNAAVVDGRLYNTYKERETLGRRVQDF
LSGKDGKGLG
>lcl|NZ_CP026520.1_prot_WP_042234386.1_5661 [locus_tag=PC41400_RS28560] [protein=hybrid sensor histidine kinase/response regulator transcription factor] [protein_id=WP_042234386.1] [location=complement(6321703..6323553)] [gbkey=CDS]
MFDAVKQWFWYEWTLFIIRILVTFSTIITTVQFQNDLNLPLWLVLCWEISAFSIPWILLQVNTKAYAVPELILSGGVSVY
LTSLFPEAYLTFLVPAALIAANNVEKTYRWTGPAAVLVVPLVMVQLSEPAKPFWEMMSQIGLAYAIGYAFNLLMVNYRQN
EIIRSQNSVLEQYMSQVERVTLLEERNRLAKDLHDTMGHSYTSLIMGLETLRPDISTEEGEHKLEALLAVTRKSMGEVRD
FLHGMGASEEMSSLRQALEQLAREFEASAGVTVSLRVLGEETALPQQAKMTFYRCLQEALTNAVRHGRAAEIQATLQFEP
RQTRLEVRDNGEGSESLSEGFGLKAMKERAASLQGQVSVYSEKGVGTTVTCTLPRQAEPGGEVIRLLIADDQAFIRESLR
TLLQGQKDMTVAGLAGDGEQAVELCAELEPDLVLMDLDMPNMDGTAATRTIKQKWPHVRVVILTTFQAPDKALEVLRSGA
DGYLLKSIEPKELAETIRLVHQGGTMIAQDLSTKLFGELGEAAKASGRGGATGTSGPAETAGRPETAAPAGEGYELTPRE
IEILRMLSKGLRYKSIASKLYLSDGTVRNYASTVYAKLGVRNREEAVEKAAETGLL
>lcl|NZ_CP026520.1_prot_WP_042234388.1_5662 [locus_tag=PC41400_RS28565] [protein=sensor domain-containing protein] [protein_id=WP_042234388.1] [location=complement(6323635..6324222)] [gbkey=CDS]
MKKEMVQSLQTFYFLLLTFATGVFYFCFYLVSLVFSLSMAFTLIGLPMITYVMRSTRTFARYERIQTKIYTDLSIEPYGE
PAIKEGSLWQQAKEELTDRRNWTIIGWLMLKFPLGLLALLLAVLCYIAPVIFILVPLAVPYLDIYFAGIRVDTFVKSLFV
MAGGVLLAYAGAKLGRSSALLVGRYIRQMFERLHT
>lcl|NZ_CP026520.1_prot_WP_042234391.1_5663 [locus_tag=PC41400_RS28570] [protein=hypothetical protein] [protein_id=WP_042234391.1] [location=6324453..6324752] [gbkey=CDS]
MPEPKHTTITQRTMRKKENEPFKKNIRHASLARLVSSRLVCLYGLYTEHADYPDITTYLGPEVYEEFMEHYSPCRERARE
AAVLLDKTFGCHTLPLKVQ
>lcl|NZ_CP026520.1_prot_WP_042234393.1_5664 [locus_tag=PC41400_RS28575] [protein=hypothetical protein] [protein_id=WP_042234393.1] [location=complement(6324739..6325434)] [gbkey=CDS]
MIRTDRSGTDFKAVPSAGLKFLELDMPVLGNLHLHFDAGEEAYVKLSGEYHLFPAYTAREGDRLVLKGRNLPAYLRHGQK
ERLRIDIHLPEDTAVDLTFLAGVVSLNGGYGPLSVKGFSGEVSGYTGSDRVRVRLRCGDVSLAGLPGEADIRIGLGSSTL
RWSRLLGTERVGVRCAFGGVELLVPPEWAQKEEQGGFFKKKRLHAPPGSSVSASVWFGGLDVGSSRGFTAP
>lcl|NZ_CP026520.1_prot_WP_042234395.1_5665 [locus_tag=PC41400_RS28580] [protein=endonuclease] [protein_id=WP_042234395.1] [location=complement(6325871..6329440)] [gbkey=CDS]
MKKRLPGWKVFKASLALSAALSVQVLTDGWQVKAEGPADPAPYIEAKVVNENAGKKVLFDNTHAQTAGAADWVIDGGFSD
FGNALANNGYYVKELRKTTPITLADLSGYDVFVVAEANIPYKASEQAAMTAYVESGGSIFFIGDHYNADRNKNRWDGSEV
FNGYRRGAWDNPAKGMSADEAGSEAMQGVVSSDWLGSQFGVRFRYNALGDITANNIVAPEQAFGITKGVSSVAMHAGSTL
AIIDPAKAKGIVYLPQTKAAWANAVDQGVYNGGGVAEGPYVAVAKKGAGKAAFIGDSSPVEDATPKYLREDTGATKKTYD
GFKEANDGVLLVNTINWLAKKESYTSLSQVSGLQLDQPTALLPFEAPAASTEPQPEPWAAPNAGYKWYDRGTFKSGSYGS
GGGTTVPSNPAYSFVRQATLPNAAEFKIRVVIDNLPANTTVSGYSAGVYLATGGTQVAQVQNADGTWPTSYGYSAAFSLT
SDSKGHAYKDLTVRIKPGTTGAANLRLRQNSNNLKTEAVSIANVPAEPLPDEQGPIPAKITVAEARGKGEGALVTLEGVV
TTEPGIFGGQSFYLQDATGGIYVYQNLGGFHAGDVVKVTASTAVYNSELELTDPVEVAKTGTSAVPTPVKADAVTAGNQG
QLIELSGVTISAVKSAVPEGSFEFDAVKGDVVNHIRVDARTGLTLASFPYKEGQTVDIRGVAAIFKGAFQLKPRGLSDFT
AQADTAAPVTEAALSETANAEGWFNKNVTVTLTARDNEGAAVLTQYDINGGEAKGYSEPFTVETEGTNIVRYYSTDAAGN
KEEAKTLRIKLDKTAPAASLTLNGQAVSDVTDKDSLTFNLLSTDAGSGIADRKLLLDGKEIAAGVTLKAGELAVGPHTVQ
YRVTDLAGNTAEDALSFNVKASLTLSKVELTSGKTVLKPGETTDLRVSGTWNNGTAADLTVAEVVYRSSNEAAVKVNAQG
RLTAVGEGTADLTASVTLNGVTVVSDAVTVTSAKPLATGVPGKPVLSSNNGYDGLKGGSFTVTMNMWWGANGSVYKLYEN
GVLISTQTLKDASPVAQTVKTDIKGKANGTYTYTAELINVYGTTVSSPLVVNVTEAAPGKPVLSQDNWDGDGTYKVTMNM
WWGTNASEYRLYENGQLIETKALKESTPGAQSAVTSLTGRAPGTYEYRAELVNASGVTSSEKMVVKVTR
>lcl|NZ_CP026520.1_prot_PC41400_RS28585_5666 [locus_tag=PC41400_RS28585] [protein=family 14 glycosylhydrolase] [pseudo=true] [partial=3'] [location=complement(<6329713..6331383)] [gbkey=CDS]
MSLTNKSLKKAALQLLGFILVLTTMFGWGPVKAGAAVAGDFQASVMGPLTKVTDWNAFKNQLRTLKNNGVYAVTTDVWWG
MVESAGDNQFDWSYYKTYAGAVREAGLKWVPILSTHRCGGNVGDDCNIPLPNWLWAKGTADQMQFKSETGYMNNEAVSPF
WSGLGTQYSELYASFAANFASYKDIIPKIYLSGGPAGELRYPSYYPAAGWSYPSRGKFQVYTDTAKQAFRTAMTAKYGSL
SGINSAWGLSLTSVDQISPPNDGDGFYTNGGYKTAYGKDFLSWYQSVLENHLGVIGSAAHKNFDSVFGVPIGAKVSGVHW
QMTNPSMPHSAEHAAGYYDYNRLLQKFKDSNLDLTFTCLEMSDSGSAPNYSLPSTLVDTVAGIASAKGVRLNGENALPAS
GTAAFQKIEEKLTRFGFSGFTLLRLANVVNADGSATGEMANFKKYVISHAGTGSGGTNNTATVYYKKGYSSPYIHYRPAG
GTWTAVPGVKMADSEISGYAKITVNIGAATQLEAAFNDGNNNWDSNSMNNYFFGTGISTYTPGTGGAAGIVKAGGPA
>lcl|NZ_CP026520.1_prot_WP_042234399.1_5667 [locus_tag=PC41400_RS28590] [protein=metallophosphoesterase] [protein_id=WP_042234399.1] [location=6331814..6332818] [gbkey=CDS]
MLRKTFLSAALSLSLLAAAVIPTAVGAKPEKEKPLLAFPVISDIHIQSWDPQSHQKFKAALSDLHTLGPDADALVINGDL
TNGMPADYAKLSELMEQNPHPRKVFYTMGNHEYYKAWIDRNGSWNADAFPNDETEQASIGRFLQLSGKKKVYGDQVVNGF
RFIFLGSEQYRQSDPSNLEDAWLSEEQLEFLKASLRKGASSGKPVFVFLHQPLPGTVSGTSFCCVNNRAIVQHAELRNIL
SAYPQAIFFSGHTHWELKLPGTLVQDKFTMVNSSSVIQPWTDNGNGGEMLTGPDESEGLYVEVYKDKVSIKGRDFYRQRW
IPEAQFTLPRSIED
>lcl|NZ_CP026520.1_prot_WP_042234402.1_5668 [locus_tag=PC41400_RS28595] [protein=fused MFS/spermidine synthase] [protein_id=WP_042234402.1] [location=6333022..6333765] [gbkey=CDS]
MQLLFEENGPGHVIRVYDTNELYGEKGRFRVLQFANDAIQGALDLNNPGRILFEYPRAIVHLMETNRPDFENVFMIGHGI
GTLAGQYPQKRFTIAEYDPKVAELSRTYFGCDHENILLGDGRELLDQEPPEAYDYVILDAFTEKGTPRHLISRQFFELSR
SKLNPDGAILMNLFGKSMHDNHISAIHTTLGEVFSYRKAFYLPSEGTRDTLNILLVGSSSPLTYQARHMAGFVEFEPERG
YLITDRL
>lcl|NZ_CP026520.1_prot_WP_042234404.1_5669 [locus_tag=PC41400_RS28600] [protein=hypothetical protein] [protein_id=WP_042234404.1] [location=6333902..6334414] [gbkey=CDS]
MKKMVFFTTLMISMLAFVTSANAVGSEYTPVPRSEDTITVMISSLENKNGSLYLQGDQIEWYEGAEADQKFLEHETDIGD
MTHAPDGYYIVNDQEKLESFEIDPNAQVLMQLYDHDGTYEGVDVKWNEEVSLTKFLAIYQNNNLLDVSTFPFHLTIKDHK
VVKIVQQYIP
>lcl|NZ_CP026520.1_prot_WP_042234406.1_5670 [locus_tag=PC41400_RS28605] [protein=GNAT family N-acetyltransferase] [protein_id=WP_042234406.1] [location=complement(6334629..6335159)] [gbkey=CDS]
MKILETDRLILRLQTTDNAAFILELLNDPSWLRFIGDRRVRTLDDARAYILNGPVRMYEQFGFCLYLVERKEDQAPIGIC
GLIKRESLEDVDIGFAFLPKYWSKGYAYEAASAVMSYGKDTLGLTRIVAITAQENQASERLLEKLGFQLERLIQLTNDSE
KLKLLVFGETENHISN
>lcl|NZ_CP026520.1_prot_PC41400_RS28610_5671 [locus_tag=PC41400_RS28610] [protein=ANT(4')-I family aminoglycoside nucleotidyltransferase] [pseudo=true] [location=complement(6335718..6336485)] [gbkey=CDS]
MYMMGPREVIREERMKIAYIIANQLM*KYGTDVEAIGIYGSLARKTDGPFSDIEIKCILNSIEDGFSYEWTSGDWKAEVN
LDNKENIVEEATTVEENWPLTHGQFFTILPIYDPKGFFQELRQKASSVDNTIFKEAICKTLVEEMYEYMGKLRNIKIQGP
ETFLPALAIEIATAGAMILGLHHKRYFTTSAQVLPEALGLSNRPDGFDKLCQLVMSGHLYDSKKILDACENFWNGLLDWS
KKNGYVINYSDSVPF
>lcl|NZ_CP026520.1_prot_WP_042234408.1_5672 [locus_tag=PC41400_RS28620] [protein=alpha-L-fucosidase] [protein_id=WP_042234408.1] [location=complement(6337026..6338300)] [gbkey=CDS]
MSTREQRTRWFLQDRFGMFIHWGLYSIPARGEWIRGNERMSYEQYQVYFDEFDASRYEPKKWAKAAKAAGQKYAVLTTKH
HDGFCLFDSKLTDFKATNTPAGRDLVREYVDAFREEGIMVGLYYSIIDWHHDDYPGYGDKAHPDRDNEAAKNKPIDFDRY
LDYMHGQVKELLTNYGKIDIMWFDFSYDDMTGEKWRATELIRMIRSIQPDIIIDNRLGGNIRAAEPEEYAGDFFSPEQII
PPGGIVDEKGEVVPWEACITLNDNWGYHAADKNYKSAKQVIRTLVECVSKNGNLLLNVAPDAKGEINRETLEVLDEVGEW
LRLNGDSIYGCKAADLPKPEWGRYTQNGNRLYAHVYDRGIGPIYFEGLKGKIEKARLLRDGTELKVEVPWMALEYSDIEN
GAFINLAGSRLPDESDTVIELHLR
>lcl|NZ_CP026520.1_prot_WP_042234410.1_5673 [locus_tag=PC41400_RS28625] [protein=ROK family protein] [protein_id=WP_042234410.1] [location=complement(6338517..6339473)] [gbkey=CDS]
MIDLQNTDKYVVGIDLGGTKIAAALFNADGEILNRERMETANARTAKEVVQRMVEMIHSVSMEHPVCGVGLASPGTVNSK
DGIVLQGTNLPEWNNIPLKKWMGDALGIEVEVVNDANAAAWGEYVRGAGKGSGNMVYITFSTGIGAGIVLDGKLLLGANS
FAGELGHNIVVPDGPACICGNNGCWEMYASGTAIRDSALELMLSNPSVIGELAAASGEEISAKHVFEAHRQDDPIAAQVV
SRAVHYMALGLANAIHTFNPDRIVIGGGVSKAGDLLFPALKAETEKLVMKPYRGTYEIVPAGLRDDVGLIGAAALFHT
>lcl|NZ_CP026520.1_prot_WP_042234412.1_5674 [locus_tag=PC41400_RS28630] [protein=copper homeostasis protein CutC] [protein_id=WP_042234412.1] [location=complement(6339816..6340508)] [gbkey=CDS]
MILEVIATCIDDALRAEENGANRLELITAVTEGGLTPGIGLIRQVVGAVQIPVHVMVRPHSRSFVYSKLDLETMKAEIQA
INEAGAAGIVVGALTAEGLIDENALGVLLPWADGMQITFHRAFDELEDQLAGLQTLMRYPQITRVLTSGGLKPATEAMPA
IRRLVEATEGRSLRILAGNGLTPEAAGSFIEQTGVTEVHFGSAVRYGRSGLGPIDPVQLRALADSLAQYS
>lcl|NZ_CP026520.1_prot_WP_241688755.1_5675 [locus_tag=PC41400_RS28635] [protein=leucyl aminopeptidase family protein] [protein_id=WP_241688755.1] [location=complement(6340505..6342145)] [gbkey=CDS]
MPKERIAMHITLDMTELADVLIHPVYAGESFPYALNEARKRQGTMTWLYARSAGEADVLAAGMGERCAVTLERVRRAGGA
AARTLLQEGRGSATLLRHPGTVVGVGGSESIESAAAWLQAWIEGWLLGLYRFDKYKNVTKISGSVKLNLRSADWPELSAA
GLNSVIQTAIHRAEATNLTRDWVNETPGTLNPDAFVEWTQDLFKDRPVNIYVYRGQELTERGMNGLLAVGAGSKHAPALI
ELRYEACPDAPMLALVGKAITFDMGGMNAKTGRDISDARMDMGGAAAVIGALDILVRQQAKVRVTALIAVADNLPDARAT
LPSSVIAYPNGLTVQIANTDGEGRLVLADALIHAAGLGAAQAIDIATLTGNVGDALGLGIAGIWGDADMTRELAQIGQHN
GERLWPMPLMDDYESELRSNYADLRNVGTSTLAGAITAALFIRRFVDKSMNWVHIDMAGTVQYKQDVGYAEAGATGYGAR
LLADYAVNYARKCDAGKAPGNGARNATEDVSEDVSENTSRNEARDAVGFVSGDVSVNESRKQEGSL
>lcl|NZ_CP026520.1_prot_WP_042234414.1_5676 [locus_tag=PC41400_RS28640] [protein=dipeptide ABC transporter ATP-binding protein] [protein_id=WP_042234414.1] [location=complement(6342163..6343149)] [gbkey=CDS]
MSKSTIETPLVEVRGLRKSFAVKKGWFGAAKYLRAVDGLSFAIRQGETFSLVGESGCGKSTTGRLVNRLLKPTSGEVWFN
GQNISSFSDNQMRPIRKEMQMVFQDPYASLNPRMKVKELVAEPLLIHTKLSVKERDKLAQELLETVGLSSIHAERYAHEF
SGGQRQRIGIARALSVRPSLIVADEPVSALDVSIQSQVLNLMQDLQEEYGLTYLFISHDLSVVEHISDRIGVMYLGTLVE
TADKDTLYDRPLHPYTQALLSSVPVPDPNQKRERIILKGDLPSPVDPPSGCRFHNRCSACMDICRQQEPVFREVEPGHQV
ACHLYNEA
>lcl|NZ_CP026520.1_prot_WP_042234416.1_5677 [locus_tag=PC41400_RS28645] [protein=ABC transporter ATP-binding protein] [protein_id=WP_042234416.1] [location=complement(6343146..6344147)] [gbkey=CDS]
MTTRLLEVEGLKTEFRRDGGSVMAVSGISFHINKGEVLGLVGESGCGKSVTSLSIMRLLKDTPGRIAGGAVRFEGTDLTQ
IQEKEMRRFRGNELAMIFQEPMTSLNPVLRIGRQLEEPIMLHLGYSRKRAREHAIHTLKLVGIPRAEDVADEYPHQLSGG
MRQRVMIAMAMSCSPKLLIADEPTTALDVTIQAQILDLMKRLKEEQDMGMLLITHDLGVVAEMCDRVVVMYAGRVVEEAP
VQELFANPQHPYTKGLIQSVPKLRQKVRRLESIQGNVPDLSAMPPGCKFMPRCAYAMERCLQEPELLPVQGEADRKSRCW
LTQMDTPEGGDRQ
>lcl|NZ_CP026520.1_prot_WP_042234418.1_5678 [locus_tag=PC41400_RS28650] [protein=ABC transporter permease subunit] [protein_id=WP_042234418.1] [location=complement(6344162..6345073)] [gbkey=CDS]
MDNNTTVMGADSSGAGVTLVTKPTKGRAAEFFSKFCKQRASLASGIFIIFLAIIAIFGPHLTPYDPNVPDYDALMAGPSA
AHWAGTDEYGRDILSRLIDGTRLTLAVSLSSVLIAAVLGTVLGLLSGYYGGWLDRLIMRGSDVLFSFPDLLLAIGIVAIL
GPGLSNVVIAVAVFGTPSFARIIRSVTLSAKETLFVEAARSMGAKNVRIIWKHIFPQTVPSVIVNVTMKIGGAILAASSL
SFLGMGAKPTEPDWGAMLSMGRDYLSIAPHIVYFPGIAIFLAVLAFNLVGDGLRDALDPKTKN
>lcl|NZ_CP026520.1_prot_WP_042234421.1_5679 [locus_tag=PC41400_RS28655] [protein=ABC transporter permease subunit] [protein_id=WP_042234421.1] [location=complement(6345086..6346012)] [gbkey=CDS]
MGRYALQRLLGVIPLLFVVSVLVFMFIHLIPGDPARLVAGKEATLEEINGIRTQLGLDLPLWQQYFHYMGKLLHGDLGSS
LRSGLPVSDMLMSRLMPTLILTFLSLGWALIIGLLIGTVSAVNRNRWPDYVGMLTAISGISIPGFWLGLVLIQIFSVQLG
WFPTGGVDSWKSYILPSLTLGAGIMSMLARFSRSSMLETMREDYIRTGRSKGLREFIVVGRHALRNSLIQVVTVAGLQFG
FLLGGSVMVETVFSIPGLGRLLVDSISFRDYTVIQALLLLFATQFILINLIVDMLYGVLNPKIRYASS
>lcl|NZ_CP026520.1_prot_WP_042234424.1_5680 [locus_tag=PC41400_RS28660] [protein=glutathione ABC transporter substrate-binding protein] [protein_id=WP_042234424.1] [location=complement(6346266..6347867)] [gbkey=CDS]
MKVQGKMWSTLMLLLIFAVVAAGCGGNTETGKESANTGKSEGTQTVSKNKKDVVIAINANFITLDPHNTSDTHSITGART
MVEGLMGFDENMKVVPVLADSYKISDDGLVYTFTLKQNIKFHDGTDFNAEAVKVNLDRIRDEKNNLKMRKSFAKVSKVEA
PDAKTVVITLSEPYNAFLNKMAMALIVSPKALKEQGENIGKNPVGTGSFKFKEWVQGDRLVVVKNPDYWQKDLPKVESIT
FKPVPENGSRIAMLKTGEADFIYPMPTEQVSQVEGDSNIVVDKTDSTIVRYVTLNTMKKPFDDVKVRQAINYAINKEAYI
KVVKSGLGVKLNSTMSSKTQYYSQQSGYDYDVKKAKNLLAEAGYPDGFTAEIWGENDSETMKGMQFIQQQLALVGIKLDV
KSMEGGTLSEQINSAATPEQAKIQMWYVSWSPSSGDADGATRGLFSSEMFPPAGANTAYYKNPNVDQWIADANKATDPEK
AKSIYADIQKTIWADAPWAFMGVDQVISGKRASLNGVKVLPDGSISVRDVEVK
>lcl|NZ_CP026520.1_prot_WP_042234427.1_5681 [locus_tag=PC41400_RS28665] [protein=asparaginase] [protein_id=WP_042234427.1] [location=complement(6347940..6348956)] [gbkey=CDS]
MRFTPLVEEYRGGVLENVHYGAVSIVDEKGRILYSAGDPEHLTFLRSAAKPFQALPVMKRKVDEVYGLTGQEAALFAASH
RGERFHIDALESMLLKTGIKEEGLHCCESYPLNEDAKAERHRDREPKRRIFHNCSGKHTGLIALSKHMGWDERTYYEPEH
PVQKEIVETLAYIAEVPEASIPQGIDGCGLPIFALPLHKIAYSFLKLACPDLIEDKETREAAARIARLMNEYPDMIADTR
FVCSELLKDPNLSAKGGAKGVYGIGLRNERLGISLKVSDGSEQVWPCIIASILERLGYSNQETINRLYKLVPNSIVNDGG
TEVGERRAVFELKPSSRE
>lcl|NZ_CP026520.1_prot_WP_042234429.1_5682 [locus_tag=PC41400_RS28670] [protein=ROK family protein] [protein_id=WP_042234429.1] [location=complement(6349014..6350171)] [gbkey=CDS]
MKKHDQDFMKRQNRLTVFEIIKHQQPISRASIAKQTGMSPTTVSRIVAELAEEGYLLESEQVSAGLGRKSTLLEMVDAAV
LSVGVELDRHQVNIGIVDLQGNVIAMSQYPRSVEESPEVTLERVGTEIDQLTKSNEIDCERIIGVGVGLPGIIDNDTGEV
MFSVQLGWRNVRLAERLKEITGYDVAVDNELKAKALAEHMKGAAIGSQRTVLLGFGNGVGSAFIMEGEIYRGRTNSAGEI
GHTTIDPNGMLCDCGKAGCLQTYINIKSLLSEANKIRPVRSIEELFEAKKAGERWALHLIERALMYMAITINNVVCMYNP
DSVILSGELTDKFPEILDEVVDLCYSKYVWEPLRGAFRILKSELHEQGVVIGSGLISQNCFFALE
>lcl|NZ_CP026520.1_prot_WP_084706677.1_5683 [locus_tag=PC41400_RS28675] [protein=VanZ family protein] [protein_id=WP_084706677.1] [location=6350777..6351082] [gbkey=CDS]
MSTSMVIASIFAAYMYVVVKVILFKLHSPEIIMDQLTVRANLTPFHEIVTYFKQATLDIWVNFAGNILIFMPFGMLLPLL
LRGMSNPWSKVFMPSFTFSIQ
>lcl|NZ_CP026520.1_prot_WP_042234436.1_5684 [gene=glmS] [locus_tag=PC41400_RS28680] [protein=glutamine--fructose-6-phosphate transaminase (isomerizing)] [protein_id=WP_042234436.1] [location=complement(6351482..6353311)] [gbkey=CDS]
MCGIVGYIGKRDSQEVLIEGLKKLEYRGYDSAGVAVYTEGGLQIKKSKGRLAVLESQLSDAPLNGTVGIGHTRWATHGKP
SDENSHPHTDNSMKFSVVHNGIIENYMTLKEELIAKGHTFVSETDTEVISHLIAELYEGDIVKAVQQAVKQMRGAFALGV
LTEYEPDRLVAVRLASPLIIGVGEGENFIGSDIPAILEYTRNVYILNDGEMALLTRDGVELMTIEGNFISREMFRVDWDL
VTAEKAGFDHFMMKEIYDQPKAYRDTIGSRISEAGDSVILDEISMTADQIRNIRNIQIVACGTAYHAGLVGKSVIEKLAR
IPVETDVASEYRYRSPIITPETLVIVVSQSGETADTLAALREAKRNGARVLAITNVVGSSVSREADDVIVTWAGPEIAVA
STKAYTTQLMAFYLLGLFLAQTLETVEPAEIRHTLAAMQSLPEQVESILARGEEIQAYAEQIAPKEHLFFIGRGIDYAVA
LEGSLKLKEISYIHSEAYPAGELKHGTLALIEDDVPVIALGLQSDLQEKTVSNVKEVTARGGKVLGIAYEGDAEFAKSVD
QTFSIPRTLDVLSPALTVIPLQLLSYYASLARGNDVDKPRNLAKSVTVE
>lcl|NZ_CP026520.1_prot_WP_042234444.1_5685 [gene=glmM] [locus_tag=PC41400_RS28685] [protein=phosphoglucosamine mutase] [protein_id=WP_042234444.1] [location=complement(6354887..6356227)] [gbkey=CDS]
MGKYFGTDGVRGVANRELTPELAFEIGRCGGYVLTRQAQHPKVVIGQDTRISGPMLEAALVAGLLSIGANVIRLGVVSTP
AVAYLTRVLGADAGVMISASHNPVEDNGIKFFGGDGFKLLDETELEIERLLDADGDELPRPVGGEIGTVSDDPKAKFLYA
DFVKSTVQHSFGGMKIVMDCANGAAYELAPKIFRELGAEVITLGAEPNGRNINEGCGSTHPENLQKAVVEHGANLGLAFD
GDADRLIAIDENGEEVDGDFILSICGDAMNRAGKLNHSTIVTTVMANIGFFKGIEGHGLKATQTAVGDRYVMEEMRKGGY
NLGGEQSGHVIFLDYNTTGDGILTGLQLADTLVAGGRPLSEVKRIMRKFPQVLVNVRVADKSKLQDNAAIAGAISAVESQ
LGDNGRVLVRPSGTESLIRVMAEGPDKDTVEGYVNDIVEVVKRELG
>lcl|NZ_CP026520.1_prot_WP_042234447.1_5686 [locus_tag=PC41400_RS28690] [protein=CdaR family protein] [protein_id=WP_042234447.1] [location=complement(6356347..6357684)] [gbkey=CDS]
MDRWLRNTNVVRVIAVIIGILLWAVVRMEGTTVTNSTTGGIKELPISNVQITPKYDSDKYYISSIEPSRVQVILKGKDSA
LKRVSTSEYKVVLDLTKAGGKGEYVLPLEAVGFPSSLQVDIVPRNVKVVVEEQEMKEVPVKINVKGTPQPGYKSGDPIAK
PSKVHVTVPASKADEVATVQGEISVDKADSAVTKQVKLKVYNKEGRELPYDIRPAVVDVDVPVTSPFKTVPLQIQLTGNL
PPGYSIGSVKQNPDKIMVYGAQNVLDGMDFYSGVTLKIDGLKETKEFTLDIPLQKGVTQIEPTKVQVTVEIVSSIKKEYP
GIPLKIVGQAEGTEVKFTDPPNGSITLTLEGAAPQMEAVKPEDIQAIVDVSNLPPGAHELPITLNLPPYIKDGGTQVTKA
KVVISAAQTQTESKNPSPTPIPTPTPTQKPDAEGETVPKKEAVTP
>lcl|NZ_CP026520.1_prot_WP_042234450.1_5687 [gene=cdaA] [locus_tag=PC41400_RS28695] [protein=diadenylate cyclase CdaA] [protein_id=WP_042234450.1] [location=complement(6357677..6358492)] [gbkey=CDS]
MELFADISITDIIDISIVSYVVYKLIMLLRGTRAVQLMKGIIVVLVAWALSVAFNLNTLQWIMNQMFTVGLIAIIIIFQP
ELRRALEQLGRGRLFSRSSMDVDQDVNIRIGEVIKAMNYLSRRKIGALIVFEKETGLTDYIESGIAIGGRISSELIINIF
IPNTPLHDGAVIIRNNQLMAAGCYLPLSENPFISKELGTRHRAAIGMSEVSDAVCLIVSEETGQISLSLNGQVVRDIKEE
SLISKLFDELKPKANDRQNPFKKWWEGRKRG
>lcl|NZ_CP026520.1_prot_WP_042234452.1_5688 [locus_tag=PC41400_RS28700] [protein=zf-HC2 domain-containing protein] [protein_id=WP_042234452.1] [location=complement(6358687..6359286)] [gbkey=CDS]
MNCKQALPLMHEYLDGDLQGAQALELKQHLLSCPECNSLFRQMETTDALVRSIPAPTVSSDLTNRIMGSLPPVKKRKPWT
QWIRRHPAVSVAAVFLLTMAGSFMTLWNNNTELMVRGADLERVIIQGDKVIVPKGETVKGNLVVENGQLVVEDDANVEGN
LVVIDGNMNLASTAHISGDKIRIDEALSWIWFKLNDLMH
>lcl|NZ_CP026520.1_prot_WP_009675029.1_5689 [gene=sigW] [locus_tag=PC41400_RS28705] [protein=RNA polymerase sigma factor SigW] [protein_id=WP_009675029.1] [location=complement(6359368..6359946)] [gbkey=CDS]
MNLVETRLAKLARNGDRGAFEELVNLYKDKIFHLAYRMLGNKQEAEDAVQDTFLRVYNNLDRYDENQKFSTWIFRIGTNL
CIDRLRKRKPTYSLDAEMPDSEGNDFYAMLASKDELPEDQIVLSETQQVIRRAIQTLPEKYKSVVILRYLHDMSLQEIGD
VLGMPVTTVKTRVHRGREYLRKKLDTDANFSL
>lcl|NZ_CP026520.1_prot_WP_042234455.1_5690 [gene=ppc] [locus_tag=PC41400_RS28710] [protein=phosphoenolpyruvate carboxylase] [protein_id=WP_042234455.1] [location=complement(6360075..6362861)] [gbkey=CDS]
MSDNETIVQKNSTNNLLRRDVRFLGHILGEVLVHQGGNELLAVVEKIRELSKSLRAQYDPAIYAEFKTTIDGLQPEIRHQ
VIRAFAIYFQLVNIAEQTHRIRRKRDYERSSGETVQPGSIESIVQQLKQNQISYQEVQEILQTISLELVMTAHPTEATRR
AVLDIHQRISQDVMYLDNPTLTYREREQLREKLLAEVMTLWQTDELRNRKPTVIDEVQNGLYYFDETLFDVLPEVYQELE
RCLDKYYPNETWHVPSFLKFGSWIGGDRDGNPSVTSHVTWETLELHRGLALKKYETILVDVMYHLSFSKNLAEISDELQK
SIEADHEVFGPHNDFWSNDDEPYRTKIAYMIEKVKNTGNTQVANFQKYTSPDEMIADLYVIERSLRSHYADYAADRYIKK
LIRQVELFGFHLATLDIRQHSREHELAMTEILNRIGIVEDYSTLQENEKIELLSKLLDEPRPITSPYMKYSESTQECLDV
YRNIYKAQHEFGRYCLNSYLISMTQGASDMLEVMVFAKEVGLYRKEADGTVVSTIQSVPLFETIDDLHAAPEIMDTLFKI
PAYRASLNSAAPMHEIMLGYSDSNKDGGVITANWELRVALRALTDTGNKNGVRLKFFHGRGGSLGRGGMPLNRSILAQPV
DTIGGGIKITEQGEVLSSRYSLQGIAYRSLEQATFALIKGAVLARNPQKHPLETEWESIMKGISEKAQNKYQDLIFRDPD
FMTFFKQSTPLPEVGELNIGSRPSKRNNSDRFEDLRAIPWVFAWTQSRYLLPAWYAAGTALQDFYQNKAENMSVLKNMYE
NWPFFKSMIDGLQMALAKADLLIAKEYGSLLKEGSIAKRIYELISAEYDLTSKLILDITGQQEILDNVPVIQESIRLRNP
YVDPLSYLQVDLIKELRGARENNEDDSFLLREVLLTINGIAAGLRNTG
>lcl|NZ_CP026520.1_prot_WP_009676794.1_5691 [locus_tag=PC41400_RS28805] [protein=hypothetical protein] [protein_id=WP_009676794.1] [location=complement(6369901..6370131)] [gbkey=CDS]
MLGFLFNDRECRELDYVLRKELDEMLFDLNDKRMDEEIKKAIESRYKVIFRMYARLASPREISKYARNRVQHQKKS
>lcl|NZ_CP026520.1_prot_WP_042235568.1_5692 [locus_tag=PC41400_RS28810] [protein=stage II sporulation protein M] [protein_id=WP_042235568.1] [location=complement(6370284..6370895)] [gbkey=CDS]
MKWRELFKHFGEMKHYLIASVLVFAAGLVLGTMYPDRFQAFLDQQMQGLEKISGLVKGKENLNLWLFIVIFLNNAIKSLM
FVYLGLFFGILPLMVLLVNGMLLGYVIQYQMEAQSLAYVLKGILPHGILEIPAIIVACAYGLRLGILVIKMIGGLFSPKR
SGNAAAEFKRVMRLTRPLILLVVGTLLLAAVVESTVTLWLLKG
>lcl|NZ_CP026520.1_prot_WP_042235565.1_5693 [gene=pdaB] [locus_tag=PC41400_RS28815] [protein=polysaccharide deacetylase family sporulation protein PdaB] [protein_id=WP_042235565.1] [location=6370996..6371778] [gbkey=CDS]
MSHFYVFNGKKIKQAVFIGVAVVFAAGIVYSERGNMAVFSSGGPAAIYNVKTEKKQIALTFDISWGETRTAPILDVLEQK
GIKNATFFLSSPWSKSHPDIVKRIIKNGYEIGSHGHKHVNYSSLGDEEIRKQIQTADGVLKEFTGKSPNLIRMPNGDFDK
RVLGIADQLGYKVIQWDTDSLDWKNPGVDTIINRVVSKAHPGDIVLLHASDSCKQTHEALPAIIDQLKSKGYEFVTVTEL
MKQAQTENKAVEEQSFKQVY
>lcl|NZ_CP026520.1_prot_WP_042235562.1_5694 [gene=gerD] [locus_tag=PC41400_RS28820] [protein=spore germination lipoprotein GerD] [protein_id=WP_042235562.1] [location=6371925..6372740] [gbkey=CDS]
MNRKTVRYVPALVAVILLASCSSNNSSQGSSSSQGNYKETKSMVLDILKSDEGRAAIDKASRSSLSSAGSGSSSVSMLTD
GQKLQMQTAVKDVLTSTESSKFVRDMMKDPRFAGDFAKVVSTQNKQLHKDLMKDPEYQKQLLQVMKNPEYQQLMMELTKT
TQYRQQMMNVIQESFQSPLFQAQMLTIMRKVMEEGIVPKGAGGQGGGQGGQGGGKGGKGGGQGGEGGQQGQGDQKNKQDQ
GQGQESDQGDQGQDEKESPNKKGKKDQDSEY
>lcl|NZ_CP026520.1_prot_WP_042235559.1_5695 [locus_tag=PC41400_RS28825] [protein=Mrp/NBP35 family ATP-binding protein] [protein_id=WP_042235559.1] [location=complement(6372763..6373872)] [gbkey=CDS]
MVTKELIRQSLEPITDPTYGASIVELNFVKDVVIKENRVSLTCVLSVTDESYKASLKERIESALSQIGAEGVHIRFREIT
DYEKNQLSAYENVSEAGSVDTAGSNAGTGQEAAPLLNPESPVNFIAVASGKGGVGKSTVTANLAVALARLGKRVGIMDAD
IYGFSIPDMMGIEEQPQLVDNRIIPVEKFGVKVISMGFFVEDNSPVIWRGPMLGKMLHNFFHEVDWGELDYLLLDLPPGT
GDIALDIHQLLPQSKEILVTTPHATAAFVAARAGAMAVHTKHEIIGVVENMAYYECSKCGEKDYVFGRGGGGMLAEQLHT
QLLAQIPLGAPDNHISEPDYSPSVYKEDSNTGQLYLELARSVVQHAQKD
>lcl|NZ_CP026520.1_prot_WP_042235556.1_5696 [gene=cwlD] [locus_tag=PC41400_RS28830] [protein=N-acetylmuramoyl-L-alanine amidase CwlD] [protein_id=WP_042235556.1] [location=complement(6373924..6374682)] [gbkey=CDS]
MRRPRKRFVVWLTWDSGIKIIMAGLLVMLMAFMYTYELPSTRTWSEWSMPLSGKTIALDAGHGGPDGGAVSQSGVIEKDI
NLAITLYLRDYLQQVGAMVVMTRETDTDLALEGTKGLSRRKTEDLLSRAEFINSKKADLFISIHLNSIPSDKWRGAQTFY
YPNHPDNATLSTLIQNELKKNLENTERVSKKADTGVYLLKTLKMPSALVEVGFLSNPEESRLLADDKYQKKVAASVYQAI
LRYFSGEKVRSS
>lcl|NZ_CP026520.1_prot_WP_042235554.1_5697 [gene=sat] [locus_tag=PC41400_RS28835] [protein=sulfate adenylyltransferase] [protein_id=WP_042235554.1] [location=complement(6374750..6375931)] [gbkey=CDS]
MPSSIAPHGGTLINRLVTGSERDRLLAKAGDLPQIRINNWAISDLDLIGVGAFSPLTGFLNEEDYLSVVERMRLADGTVW
SIPVTLAVEEAVAEELAVGKQAALVGEEDGVVYGLIDVTSIYKADQQLEARKVFKTDDTEHPGVQKLLTKPSTYVGGPVQ
VLNRPKPAKFEDFYYDPAETRRIFDEKNWRTIVGFQTRNPVHRAHEYIQKSAMEIVDALFLNPLVGETKSDDISADVRMK
SYLVLLENYYPSDRAFLGVFPAAMRYAGPREAIFHALVRKNYGCTHFIVGRDHAGVGDYYGTYEAQEIFSNFTPEEIGIT
PLFFEHSFYCTKCGNMASSKTCPHDKADHLHLSGTKVRALLRDGQCPPQEFTRPEVAKVLIDGMSEPVGTSAQ
>lcl|NZ_CP026520.1_prot_WP_042235552.1_5698 [locus_tag=PC41400_RS28840] [protein=phosphoadenylyl-sulfate reductase] [protein_id=WP_042235552.1] [location=complement(6375974..6376666)] [gbkey=CDS]
MNLFEKEVLVNKAAEEFENQSPEELIRWAVNTFPSITLACSFGAEDVVLVDMLQKVSPETDIFYLDTNVHFQETYDTRDR
LQDHYQKSFIQVLPKLTLEEQAAQYGAELWLSTPNECCNIRKVDPLTDILKNYDAWITGIRRDQAPTRANAKKVEYDVKF
GLVKFNPLAGWTSDDVWEYIRSNGIIYNALHDQNYPSIGCSHCTKAVTPGEDPRAGRWAGNDKTECGLHK
>lcl|NZ_CP026520.1_prot_WP_009676832.1_5699 [gene=rpsI] [locus_tag=PC41400_RS28845] [protein=30S ribosomal protein S9] [protein_id=WP_009676832.1] [location=complement(6376948..6377340)] [gbkey=CDS]
MAQVQYYGTGRRKHSVARVRLVPGEGRIIINKRDLNEYFGLETLKLIVKQPLTLTETIGKYDVLVLANGGGMSGQAGAIR
HGISRALLKADPEFRGSLKRAGFLTRDPRMKERKKYGLKAARRAPQFSKR
>lcl|NZ_CP026520.1_prot_WP_042235549.1_5700 [gene=rplM] [locus_tag=PC41400_RS28850] [protein=50S ribosomal protein L13] [protein_id=WP_042235549.1] [location=complement(6377361..6377798)] [gbkey=CDS]
MRTTYMAKPNEVERKWYIIDATGKTLGRLASEAASIIRGKHKPQFTPHVDSGDFVIIINAEKIHLTGNKLQDKKYYRHSM
HPGGLKVTTAQDMLNKKPERVLELAIHGMLPKTRLGNSMKLKLKVYAGAEHPHQAQQPEVWELRG
>lcl|NZ_CP026520.1_prot_WP_009676806.1_5701 [gene=truA] [locus_tag=PC41400_RS28855] [protein=tRNA pseudouridine(38-40) synthase TruA] [protein_id=WP_009676806.1] [location=complement(6377973..6378743)] [gbkey=CDS]
MRNIRITVSYDGTSYSGFQTQPHGNTVQDYLERAIRMLSGETVKITSSGRTDAGVHARAQVINFTTASQIPAERWCLALN
SRLPKDIVALRADEVPLSFHARKSAKRKTYSYTIRTARIPDTFQRRYQFHHPNPLDVDAMRDALAAILGTHDFSSFCSVK
TVAESRIRTIYDAWIVEETDQMSVEPNQPGQLRFYFTGSGFLYNMVRILVGTLIQIGEGKRPAEEMAAILRGRNRALAGP
TAEAHGLMLWNVEYAE
>lcl|NZ_CP026520.1_prot_WP_009676799.1_5702 [gene=rplQ] [locus_tag=PC41400_RS28860] [protein=50S ribosomal protein L17] [protein_id=WP_009676799.1] [location=complement(6378867..6379229)] [gbkey=CDS]
MAYQKLGRDSSARKALFRDLVTDLFINERIQTTEAKAKEVRSIAEKLITLAKRGDLHARRQVAAFVRRETAGDQDAIQKL
FSELATRYAERPGGYTRILKLGPRRGDAAPMVYLELVDRA
>lcl|NZ_CP026520.1_prot_WP_009676831.1_5703 [locus_tag=PC41400_RS28865] [protein=DNA-directed RNA polymerase subunit alpha] [protein_id=WP_009676831.1] [location=complement(6379254..6380198)] [gbkey=CDS]
MIEIEKPKIETVSLSEDGAYGKFVVEPLERGYGTTLGNSLRRILLSSLPGAAVSSVQIDGVLHEFSTIPGVMEDVTEIIL
NLKGLSLKIHSDEEKVLEIDAEGEGRITAADIRGDSDVEILNPDLHIATLSSDSRLHMRIHANRGRGYVPADKNKRDEQL
IGVIPVDSIYTPITRVNYTVENTRVGQVTNYDKLTLEVWSDGSIRPEEAVSLGAKILTEHLMLFVGLTDEAKDAEIMVEK
EEDKKEKVLEMTIEELDLSVRSYNCLKRAGINTVQELITKTEEDMMKVRNLGRKSLEEVQEKLEELGLGLRTEE
>lcl|NZ_CP026520.1_prot_WP_016423660.1_5704 [gene=rpsK] [locus_tag=PC41400_RS28870] [protein=30S ribosomal protein S11] [protein_id=WP_016423660.1] [location=complement(6380321..6380716)] [gbkey=CDS]
MAKPKKVVRTKRRDRKNIESGVAHIRSTFNNTIVTITDPHGNALSWASAGNLGFKGSRKSTPFAAQMAAESAARAAMEHG
LKVVEVMVKGPGAGREAAIRSLQAAGLEVNLIKDVTPIPHNGCRPPKRRRV
>lcl|NZ_CP026520.1_prot_WP_016423661.1_5705 [gene=rpsM] [locus_tag=PC41400_RS28875] [protein=30S ribosomal protein S13] [protein_id=WP_016423661.1] [location=complement(6380738..6381106)] [gbkey=CDS]
MARIAGVDLPRDKRVEIALTYIFGIGKTTSQKVLSQTGVDASTRVRDLTEEEVSKIREVIDKTIKVEGDLRREIALNIKR
LVEIGCYRGIRHRRGLPVRGQRTKTNARTRKGPRRTVANKKK
>lcl|NZ_CP026520.1_prot_WP_003333770.1_5706 [gene=rpmJ] [locus_tag=PC41400_RS28880] [protein=50S ribosomal protein L36] [protein_id=WP_003333770.1] [location=complement(6381127..6381240)] [gbkey=CDS]
MKVRPSVKPICEKCKVIRRKGNVMVICENPKHKQKQG
>lcl|NZ_CP026520.1_prot_WP_009676800.1_5707 [gene=infA] [locus_tag=PC41400_RS28885] [protein=translation initiation factor IF-1] [protein_id=WP_009676800.1] [location=complement(6381329..6381547)] [gbkey=CDS]
MAKEDVIEVEGTVIEPLPNAMFKVELENGHKILAHVSGKIRMHFIRILPGDKVTVELSPYDLTRGRITYRFK
>lcl|NZ_CP026520.1_prot_WP_042235574.1_5708 [locus_tag=PC41400_RS28890] [protein=KOW domain-containing RNA-binding protein] [protein_id=WP_042235574.1] [location=complement(6381551..6381862)] [gbkey=CDS]
MVGQEQLPQPGQIAKILRGKDQGQYAVIIRAEDSRFVWIADGDKRRFDQPKKKNVQHLQLQPEISGEVSRSLTESSRVTN
AKLRHAVKKFIDSLQSEADEKGE
>lcl|NZ_CP026520.1_prot_WP_042235545.1_5709 [gene=map] [locus_tag=PC41400_RS28895] [protein=type I methionyl aminopeptidase] [protein_id=WP_042235545.1] [location=complement(6381872..6382624)] [gbkey=CDS]
MIICKSEVELELMREAGRIVAETHKLMAESVRPGVTTKDLDKIAEEYIRSQGAVPSFKGYNGFSGSICTSVNEELVHGIP
GSRKLVEGDVISIDIGAQYKGYHGDSAWTYGVGEISETAKKLLEVTERSLFAGLAEAKPDARLYTISHAIQQVIEDAGFS
VVREYVGHGIGTELHEDPQIPNYGIRDKGPRLKPGMVLAIEPMVNVGQRFVKTLEDDWTVVTVDNSLCAHFEHTIAITAD
GYEILTQLKA
>lcl|NZ_CP026520.1_prot_WP_042235542.1_5710 [locus_tag=PC41400_RS28900] [protein=adenylate kinase] [protein_id=WP_042235542.1] [location=complement(6382621..6383274)] [gbkey=CDS]
MNVLFMGPPGAGKGTQAEKIVSEFQIPHISTGDAFRLAMSQGTPLGIEAKSYVDQGKLVPDEITNGIVRERLQQSDCDKG
FLLDGFPRTIAQAEALDEILQSLGKRIDHVINLKVDRSFLLARLTGRRICKSCGATYHVIFNPPKQEGVCDKCSGELYQR
SDDTEEKVGTRLDEYTSKTAPLLEYYQKRNVLREVDGEQDIEDVTSQISSLLRGHSE
>lcl|NZ_CP026520.1_prot_WP_042235539.1_5711 [gene=secY] [locus_tag=PC41400_RS28905] [protein=preprotein translocase subunit SecY] [protein_id=WP_042235539.1] [location=complement(6383360..6384667)] [gbkey=CDS]
MFRTLSNIFKVEDLRKRIIFTLLVLIIYRIGAFIPVPNINAELLKLQDQANQNSVFGLLNTFSGGALFQFSIFSMSIFPY
ITASIIVQLLSMDVVPRLTQWAKEGEVGRKKSAQLTRYLTVVLGVIQAYGYAIMFNNSFQGLVINPSFATYSLIAIVLTA
GTMFLMWLGEQITEKGIGNGISIIIFAGIVAAIPTGISQIYQTQFANAGDALFLNILKVVLILIVVVAIIVGVIYVQQGV
RKIPVQYAKRVVGRKMYGGQSTHIPMKVNAAGVIPVIFAVSLLQFPTIIAGFWAGNPFADWIIRNLSINNSAPIALALYV
LLIVGFTYFYTFVQINPVQMADQMKKNGGYIPGVRPGKPTSSYLTRVMSRITLSGAFFLAFISILPMIFGALAGLPTSVR
IGGTSLLILVGVALETLKTVESQLIKRHYKGFINK
>lcl|NZ_CP026520.1_prot_WP_009676816.1_5712 [gene=rplO] [locus_tag=PC41400_RS28910] [protein=50S ribosomal protein L15] [protein_id=WP_009676816.1] [location=complement(6384667..6385107)] [gbkey=CDS]
MKLHELSSAEGARHTRKRLGRGVGSGLGKTSGRGHKGQNARSGGGVRPGFEGGQNPLYRRLPKRGFVNPTRKEFAIVNLE
DLNKFEAGTEVTPELLMETGVVKSPKDGIKILGNGELTVKLTVKANKFSQSAVDKIQAAGGQSEVI
>lcl|NZ_CP026520.1_prot_WP_009676811.1_5713 [gene=rpmD] [locus_tag=PC41400_RS28915] [protein=50S ribosomal protein L30] [protein_id=WP_009676811.1] [location=complement(6385139..6385324)] [gbkey=CDS]
MSKLQITLKRSLIGRPETQKKTVQALGLRKLNHSVVQSDNEAIRGMINQVSHLIEVKEIQA
>lcl|NZ_CP026520.1_prot_WP_009676829.1_5714 [gene=rpsE] [locus_tag=PC41400_RS28920] [protein=30S ribosomal protein S5] [protein_id=WP_009676829.1] [location=complement(6385336..6385833)] [gbkey=CDS]
MRVDPNTLELTEKVVHINRVAKVVKGGRRFSFSALVVVGDGNGYVGAGIGKASEVPDAIRKGIEDAKKNLVHVPIVGTTI
PHLVTGHFGAGRVLLKPASEGTGVIAGGPVRAVLELAGVGDILTKSLGSSNSMNMVNATLEGLSRLKKAEEVAKLRGKTV
EELMG
>lcl|NZ_CP026520.1_prot_WP_042235537.1_5715 [gene=rplR] [locus_tag=PC41400_RS28925] [protein=50S ribosomal protein L18] [protein_id=WP_042235537.1] [location=complement(6385862..6386227)] [gbkey=CDS]
MITKGDKNKARLKRHLRVRKKIQGTTARPRLNIFRSEKHIYAQIIDDVNGVTVASASTQDKELKEVSNGGNVEAARKVGE
LVANRAKEKGVSQVVFDRGGYLYHGRVQALAEAAREAGLEF
>lcl|NZ_CP026520.1_prot_WP_009676801.1_5716 [gene=rplF] [locus_tag=PC41400_RS28930] [protein=50S ribosomal protein L6] [protein_id=WP_009676801.1] [location=complement(6386280..6386822)] [gbkey=CDS]
MSRIGRKPITVPSGVDVKLENTTITVKGPKGTLSRELHKDMKVSVQDNVITVERPSDHKLHRSLHGTTRSVLNNMVTGVT
QGFEKNLELVGVGYRANKTGNKLVLNVGYSHPVEINPEGGIEFDVPSNTKVIVKGIDKELVGATAAKIRAVREPEPYKGK
GIKYEGERIIRKEGKAGKKK
>lcl|NZ_CP026520.1_prot_WP_009676818.1_5717 [gene=rpsH] [locus_tag=PC41400_RS28935] [protein=30S ribosomal protein S8] [protein_id=WP_009676818.1] [location=complement(6386852..6387250)] [gbkey=CDS]
MVMSDPIADMLTRIRNANIVRHETVEIPASKIKKEIAEILKNEGFIRDAEFVEDNKQGIIRLFLKYGPNNERVITGLKRI
SKPGLRVYTKSQEIPRVLGGLGIAIISTSGGVMTDKDARQAKSGGEVLCYVW
>lcl|NZ_CP026520.1_prot_WP_040953112.1_5718 [locus_tag=PC41400_RS28940] [protein=type Z 30S ribosomal protein S14] [protein_id=WP_040953112.1] [location=complement(6387283..6387468)] [gbkey=CDS]
MAKTSMKIKQQRPPKFKVQAYTRCERCGRPHSVLRKFKICRICFRELAYKGQIPGVRKASW
>lcl|NZ_CP026520.1_prot_WP_042235534.1_5719 [gene=rplE] [locus_tag=PC41400_RS28945] [protein=50S ribosomal protein L5] [protein_id=WP_042235534.1] [location=complement(6387562..6388104)] [gbkey=CDS]
MAARMKDRFLNEITPALMQKFNYTTVMQVPKIEKVVINMGVGEAVSNSKVLDSAVEDLQKIAGQKPVITRAKKSIAGFKL
RENMPIGVKVTLRGQRMYQFLDKLFNVSLPRVRDFRGVSSKAFDGRGNYTLGLKEQLIFPEIEYDQIDKVRGMDIVIVTT
ANTDEEARELLTQLGMPFVK
>lcl|NZ_CP026520.1_prot_WP_009676802.1_5720 [gene=rplX] [locus_tag=PC41400_RS28950] [protein=50S ribosomal protein L24] [protein_id=WP_009676802.1] [location=complement(6388134..6388487)] [gbkey=CDS]
MPRTKKRLESHNHKLHVKKDDTVFVISGKDKGKKGRVLAAYPRENRVLVEGVNMIKKHAKPSQQNPQGGIITQEAAIHAS
NVMLVDPKSGKPTRVGYKVLENGTKVRIAKKSGEAID
>lcl|NZ_CP026520.1_prot_WP_009676804.1_5721 [gene=rplN] [locus_tag=PC41400_RS28955] [protein=50S ribosomal protein L14] [protein_id=WP_009676804.1] [location=complement(6388589..6388957)] [gbkey=CDS]
MIQPLSRLKVADNSGAKQLMCIRVLGGTGRRVGNIGDLIICSVKEATPGGVVKKGDVVKAVIVRTKSGARRKDGSYITFD
ENAAVIVKDDKSPRGTRIFGPVARELRDKDFMKIVSLAPEVI
>lcl|NZ_CP026520.1_prot_WP_009676813.1_5722 [gene=rpsQ] [locus_tag=PC41400_RS28960] [protein=30S ribosomal protein S17] [protein_id=WP_009676813.1] [location=complement(6388990..6389253)] [gbkey=CDS]
MAERNERKVQTGKVVSDKMDKTIVVAVETYKKHDLYHKRIKYTKKFKAHDENNEAQIGDTVKIMETRPLSKDKRFRLVEI
VEKAVIV
>lcl|NZ_CP026520.1_prot_WP_009676819.1_5723 [gene=rpmC] [locus_tag=PC41400_RS28965] [protein=50S ribosomal protein L29] [protein_id=WP_009676819.1] [location=complement(6389280..6389477)] [gbkey=CDS]
MKASEFRNLTTAEIEQRINGFKEELFNLRFQLATGQLDNPVQIQKLRKEIARAKTVLRERELGIG
>lcl|NZ_CP026520.1_prot_WP_009676823.1_5724 [gene=rplP] [locus_tag=PC41400_RS28970] [protein=50S ribosomal protein L16] [protein_id=WP_009676823.1] [location=complement(6389467..6389901)] [gbkey=CDS]
MLVPKRVKHRKEHRGKMRGRAKGGTEVVFGEYGLQALEPAWITNRQIEAARIAMTRYIKRGGKVWIKIFPSKPITQKPLE
VRMGSGKGNVEKWVSVVKPGKVMFELAGVSEEVAREAMRLAMHKLPVKCKFVKKEEVGGEAHEG
>lcl|NZ_CP026520.1_prot_WP_042235530.1_5725 [gene=rpsC] [locus_tag=PC41400_RS28975] [protein=30S ribosomal protein S3] [protein_id=WP_042235530.1] [location=complement(6389904..6390566)] [gbkey=CDS]
MGQKVSPVGLRIGIIRDWESKWYAEKDFGDLLMEDVKIREYLKNKLKDSAVSHIDIERAANRVNVTIHTAKPGMVIGKGG
AEVENIRNYISNMTGKKVHINIAEVKNPELDAVLVAESIAQQLERRVSFRRAMKQAMQRTQRAGAKGIKVSTSGRLGGAE
IARTEGYSEGTVPLHTLRADIDYGTAEAHTTYGRIGVKVWIYRGEVLPTAKKKAQSEGGN
>lcl|NZ_CP026520.1_prot_WP_016423663.1_5726 [gene=rplV] [locus_tag=PC41400_RS28980] [protein=50S ribosomal protein L22] [protein_id=WP_016423663.1] [location=complement(6390580..6390912)] [gbkey=CDS]
MQAKAAAKFVRIAPRKVQLVIDLIRGKQVGDAVAILRHTPKSASPIVEKVLNSAIANAEHNYSMDPNKLVISEVYVNHGP
IMKRFRPRAMGRASRINKRTSHITVVVSEK
>lcl|NZ_CP026520.1_prot_WP_042235527.1_5727 [gene=rpsS] [locus_tag=PC41400_RS28985] [protein=30S ribosomal protein S19] [protein_id=WP_042235527.1] [location=complement(6390943..6391221)] [gbkey=CDS]
MGRSLKKGPFIDGHLMKKVEELNESSKKTVVKTWSRRSTIFPQFVGHTIAVYDGRKHVPVYVTEDMVGHKLGEFAPTRTY
KGHDDDKKTGKR
>lcl|NZ_CP026520.1_prot_WP_042235524.1_5728 [gene=rplB] [locus_tag=PC41400_RS28990] [protein=50S ribosomal protein L2] [protein_id=WP_042235524.1] [location=complement(6391270..6392100)] [gbkey=CDS]
MPIKKYKPTSPARRAMSVSSFEEITTSTPEKSLLAPLSKQAGRNNQGKITTRHHGGGHKRKYRIIDFKRNKDGIPGRVAT
IEYDPNRSANIALIHYADGEKRYIIAPKGLKVDDQIQSGLGSDIKVGNALALENIPVGTVIHNIELKPGKGGQLVRAAGT
EAQLLGKEDVYVTIRLSSGEVRKVLKGCRATIGSVGNEDHELVKIGKAGRNRWLGKRPEVRGVVMNPNDHPHGGGEGRAP
IGRKSPLSPWGKPTLGYKTRKKGKASDKYIVRRRTK
>lcl|NZ_CP026520.1_prot_WP_042235521.1_5729 [gene=rplW] [locus_tag=PC41400_RS28995] [protein=50S ribosomal protein L23] [protein_id=WP_042235521.1] [location=complement(6392131..6392421)] [gbkey=CDS]
MKNPRDLIKRPVITERSTEMMADKKYVFEVEIRANKTEIKQAIEQIFKVKVSSVNTLRMPAKPKRQGRHSGYTSEWKKAI
VTLSADSKELEFFETV
>lcl|NZ_CP026520.1_prot_WP_042235518.1_5730 [gene=rplD] [locus_tag=PC41400_RS29000] [protein=50S ribosomal protein L4] [protein_id=WP_042235518.1] [location=complement(6392418..6393044)] [gbkey=CDS]
MPKVALYNVSGTQVGEVELSDVVFGIEPHVHALHEAVLLQQASVRQGTHKVKGRSEVRGGGRKPWKQKGTGRARQGSIRA
PQWKGGGVVFGPTPRSYGYKLPKKVRRLAIKSALSSKVIDNDIIVLDQLQMNAPKTKEFVAILKNLNVERKALVVAASYE
DSVALSARNIPGVKFVVADGINVLDVMVHDKLIITKDAVQKVEEVLAQ
>lcl|NZ_CP026520.1_prot_WP_042235516.1_5731 [gene=rplC] [locus_tag=PC41400_RS29005] [protein=50S ribosomal protein L3] [protein_id=WP_042235516.1] [location=complement(6393061..6393684)] [gbkey=CDS]
MKGILGKKLGMTQVFTPEGNVVPVTVIQAGPCVVLQKKDVDNDGYESIQIGFADKKEKNATKPEIGHAKKANTAPKRYVK
EFRGVALGEYEVGQELNAALFSEGEFVDVSGTSKGKGFQGVIKRHNQHTGPMSHGSRYHRGPGSMGSIAANRVMKGKKLP
GHMGSETVTLQNLQVIKVDAERNVILVKGSIPGPRNSYVSVKSAVKK
>lcl|NZ_CP026520.1_prot_WP_009676797.1_5732 [gene=rpsJ] [locus_tag=PC41400_RS29010] [protein=30S ribosomal protein S10] [protein_id=WP_009676797.1] [location=complement(6393730..6394038)] [gbkey=CDS]
MAKQKIRIRLKAYDHRVIDQSAEKIVETAKRSGASVSGPIPLPTEKQIITILRAVHKYKDSREQFEQRTHKRLIDIVNPT
PQTVDALMRLDLPSGVDIEIKL
>lcl|NZ_CP026520.1_prot_WP_042235513.1_5733 [gene=tuf] [locus_tag=PC41400_RS29015] [protein=elongation factor Tu] [protein_id=WP_042235513.1] [location=complement(6394470..6395660)] [gbkey=CDS]
MAKAKFERNKPHVNIGTIGHVDHGKTTLTAAITNVLSKRYGGSAVAFDQIDKAPEERERGITISTAHVEYETPNRHYAHV
DCPGHADYVKNMITGAAQMDGAILVVSAADGPMPQTREHILLSRQVGVPYIVVFMNKCDMVEDEELLELVEMEIRDLLSE
YEFPGDDTPIIRGSAREALQNPDGEWADKIIELFEQVDTYIPTPERETDKPFLMPVEDVFSITGRGTVATGRVERGTVKV
QDEVEIVGIAEETRKCVVTGVEMFRKLLDSAQAGDNIGALLRGVDRKDIERGQVLSKPGSVNPHTNFSAQIYVLTKEEGG
RHKPFFTGYRPQFYFRTTDVTGIIALPEGTEMVMPGDNITVTVELINPIAIEQGTKFAIREGGRTVGAGAVATISK
>lcl|NZ_CP026520.1_prot_WP_042235509.1_5734 [gene=fusA] [locus_tag=PC41400_RS29020] [protein=elongation factor G] [protein_id=WP_042235509.1] [location=complement(6395706..6397781)] [gbkey=CDS]
MAREFSLKDTRNIGIMAHIDAGKTTTTERILYYTGRTHKIGEVHEGAATMDWMEQEQERGITITSAATTAAWKNHRINII
DTPGHVDFTVEVERSLRVLDGAVGVFSAKEGVEPQSETVWRQADKYSVPRIAYVNKMDIIGADFLGVVDSMRDKLGANAV
AIQLPIGAENDFVGIIDIVEEKAILYKDDLGKEVEDIEIPAEYKDKVAELRMELVEKVAELDEELTMKYLEGEEITVAEL
KAALRKGVVSVQIFPVICGSSYRNKGVQPMLDAVIDYLPAPVDVPDIQGVLEDGTETIRKSSDSEPFAALAFKIMTDPYV
GKLTFFRVYSGVLQSGSYVLNATKGKRERVGRILQMHANSRQEISEVYAGDIAAAVGLKDTGTGDTLCDEKNAVILESMN
FPDPVIEIAVEPKTKADQDKMGVALSKLTEEDPTLRASTNEETGQTILAGMGELHLEIIVDRMRREFKVDTNVGKPQVAY
RETFKSGAKVEGKFVRQSGGRGQYGHCWVEFQPLEPGSGFVFESKIVGGAIPREYIAPIQAGIEESMKNGVIAGFPVVDI
KATVVDGSYHDVDSNEMAFKIAGSMALKAAKDKCSPALLEPIMKVEVTVPEEYMGDVMGDLNSRRGRIEGMDNRSGAQII
RAKVPLSEMFGYSTTLRSRTQGRGVYSMEISHYEEVPRTIAEEIIAKNKGA
>lcl|NZ_CP026520.1_prot_WP_042235507.1_5735 [gene=rpsG] [locus_tag=PC41400_RS29025] [protein=30S ribosomal protein S7] [protein_id=WP_042235507.1] [location=complement(6397817..6398287)] [gbkey=CDS]
MPRKGPVTRRDVLPDPIYNSKLVTRLINRIMIDGKRGVAQSILYNAFELVQERTGKEPMEVFEQAIKNIMPVLEVKSRRV
GGANYQVPIEVKPERRTTLGLRWLVNYSRLRGEKTMEERLAFEIIDASNNTGSSVKKREDIHKMAEANKAFAHYRW
>lcl|NZ_CP026520.1_prot_WP_042235505.1_5736 [gene=rpsL] [locus_tag=PC41400_RS29030] [protein=30S ribosomal protein S12] [protein_id=WP_042235505.1] [location=complement(6398330..6398749)] [gbkey=CDS]
MPTINQLVRKGRQAKVVKSKSPALQRGFNALKRESTEISAPQKRGVCTRVGTMTPKKPNSALRKYARVRLTNRVEVTAYI
PGIGHNLQEHSVVLIRGGRVKDLPGVRYHIVRGALDTSGVNNRMQARSKYGTKRPKVKK
>lcl|NZ_CP026520.1_prot_WP_009674511.1_5737 [locus_tag=PC41400_RS29035] [protein=ribosomal L7Ae/L30e/S12e/Gadd45 family protein] [protein_id=WP_009674511.1] [location=complement(6398908..6399159)] [gbkey=CDS]
MTYDKVKQASKLSVGTKQATKSVELGQALEVIVAKDADPRLTTKIVNLCSKSGVEVTYVDSMKLLGKACGIEVGAAVAAI
VDK
>lcl|NZ_CP026520.1_prot_WP_042235503.1_5738 [gene=rpoC] [locus_tag=PC41400_RS29040] [protein=DNA-directed RNA polymerase subunit beta'] [protein_id=WP_042235503.1] [location=complement(6399318..6402938)] [gbkey=CDS]
MLDVNNFEFMKIGLASPEKIRSWSRGEVKKPETINYRTLKPEKEGLFCEKIFGPTKDWECHCGKYKRVRYKGVVCDRCGV
EVTRQKVRRERMGHIELAAPVSHIWYFKGIPSRMGLALDMSPRSLEEIIYFASYVVTDPGDTPLEKKQLLSEKEYRSYRE
KYGYAFQAGMGAEAVKKLLQDIDIDKDVESLKEELKTAQGQRRNRAIKRLEVMEAFRNSKNRPDWMVLDVLPVIPPELRP
MVQLDGGRFATSDLNDLYRRVINRNNRLKRLLDLGAPDIIVQNEKRMLQEAVDALIDNGRRGRPVTGPGNRPLKSLSHML
KGKQGRFRQNLLGKRVDYSGRSVIVVGPSLKMYQCGLPKEMALELFKPFVMKELVNKGLAHNIKSAKRKVERVSPEVWDV
LEEVIKEHPVLLNRAPTLHRLGIQAFEPILVEGRAIKLHPLVCTAYNADFDGDQMAVHVPLSAEAQAEARLLMLAAGNIL
NPKDGKPVVTPSQDMVLGSFYLTTDNKHGRGANLIFRSVNEAVSAYQNGTAELHARVAIPAKVLNKKNFTEAQQNAYLIT
TIGKIIFNEIFPDDFPYINDATKANLLNGPSDDYFVFEKGSNIRARMEELPEAKAVGKDYLGSIIAECFRNYHTTRTSVI
LDKIKELGFTYSTKAGITVAVADVVVPNEKKKILEDSEEKVRTVTNQYRRGLITDDERYDRVIAIWSKTKDEITDILMRS
LDKYNSINMMVESKARGNKSQITQLGGMRGLMANPSGKIIELPIKSNFREGLTVLEYFISTHGARKGLADTALRTADSGY
LTRRLVDVAQDVIVREDDCGTDKGFIVSKIQDGKEVIEDLFDRIEGRYAFETIRHPETGEVIVNRNELIEANIANAIIDA
NIERLQIRSVLSCRSRHGVCKKCYGRNLATGQHVEIGEAVGIIAAQSIGEPGTQLTMRTFHTGGVAGDDITQGLPRIQEL
FEARNPKGQAIISEVDGLVKEIREAKDRREIEVQGEAESRVYAVPYGSRVRAVVGQTIEAGDELTEGSIDPKEMLRIKGI
RGVQNYILQEVQRVYRNQGVEINDKHVEVMIRQMLRKIRIVDAGDTTLLPGSFVDMHEYEEANRVALFADKEPAVAKPIL
LGITKASLETDSFLSAASFQETTRVLTDAAIKGKVDQLLGLKENVIIGKLIPAGTGMTRYRNIRVVDPNQEPSEEENTEK
EAVSVE
>lcl|NZ_CP026520.1_prot_WP_042235501.1_5739 [gene=rpoB] [locus_tag=PC41400_RS29045] [protein=DNA-directed RNA polymerase subunit beta] [protein_id=WP_042235501.1] [location=complement(6403017..6406553)] [gbkey=CDS]
MAGHLVQFGRRKRRTYARINEVLDIPNLIEIQQKSYEWFLDEGLREMFLDISPIQDFTGNLVLEFIDYSLGEPKYSVDES
KERDVTYAAPLRVKVRLINKETGEVKEQEVFMGDFPIMTETGTFIINGAERVIVSQLVRSPSVYYSTKVDKNGKKTYTAT
VIPNRGAWLELETDAKDIIYVRIDRTRKIPVTVLLRALGFGTDAEILDLLGDNEYIRNTLDKDNTDSTDKALIEIYERLR
PGEPPTLDNARSLLVARFFDPKRYDLANVGRYKINKKLHIKNRLFNQRLAETLADPETGEIVAEAGQLLDRRLLDEVLPF
LEKKIGFKDYSVPNGVAGEEIIPVQTIDVYSPLEDGKVVKVISNAVIDKSVKNITPADIISSINYFINLLHGIGNTDDID
HLGNRRLRSVGELLQNQFRIGLSRMERVVRERMSIQDANVITPQALINIRPVIAAIKEFFGSSQLSQFMDQTNPLAELTH
KRRLSALGPGGLTRERAGFEVRDVHHSHYGRMCPIETPEGPNIGLINSLSTFARINEYGFIEAPYRHVDPKTGVVTEEIV
YMTADEEDNYVVAQANAELTEEGTFVNDSVIVRYKDDILTLPKERVDYMDVSPKQVVSVATALIPFLENDDSNRALMGSN
MQRQAVPLLIPKAPLVGTGMEHKSAKDSGVCIVAKHDGIIEKVAGNEIWLRRQELVDGKLVNGNLIKHKLHKFMRSNQGT
CINQRPIVQKGQLVKAGDILADGPSTEQGELALGRNVVVAFMTWEGYNYEDAILLSERLVKEDVYTSIHIEEYESEARDT
KLGPEEITRDIPNVGEDALKNLDERGIIRVGAEIGAGDILVGKVTPKGVTELTAEERLLHAIFGEKAREVRDTSLRVPHG
TDGIVVDVKVFTRENGDELPPGVNQLVRVYIAQKRKISEGDKMAGRHGNKGVIARILPEEDMPFMPDGTPVEVVLNPLGV
PSRMNIGQVLEVHLGMASRALGIHVATPVFDGAREYDVFDTMEEAGMQRNGKTKLYDGRTGDAFEREVTVGVMYMIKLAH
MVDDKIHARSTGPYSLVTQQPLGGKAQFGGQRFGEMEVWALEAYGAAYTLQEILTVKSDDVVGRVKTYESIVKGENVPEP
GVPESFKVLIKELQSLGMDVKILSGDEEEIEMKELDDEEDTTSDKLNLNLEGAEIGVE
>lcl|NZ_CP026520.1_prot_WP_042235499.1_5740 [locus_tag=PC41400_RS29050] [protein=class I SAM-dependent methyltransferase] [protein_id=WP_042235499.1] [location=complement(6406973..6407572)] [gbkey=CDS]
MTDHYYTQKPTVKHDVREIEFQVGDRKLRLMTDAGVFSKNEVDFGSRLLIETMVLPAKARVLDVGCGYGPIGLSAAFKYP
SGHVTMVDINERAVELSRQNAERNGIRNVSVLQSDKLEAVKTEKFDVVLTNPPIRAGKQTVHAIFEEAYECLNPGGALWV
VIQKKQGSPSALAKMEELFGEAEEVTKDKGYRILRAFKK
>lcl|NZ_CP026520.1_prot_WP_009674515.1_5741 [gene=rplL] [locus_tag=PC41400_RS29055] [protein=50S ribosomal protein L7/L12] [protein_id=WP_009674515.1] [location=complement(6407723..6408088)] [gbkey=CDS]
MSNEQILEAIKGMTVLELNDLVKAIEEEFGVTAAAPVAVMGGAAGGAEAAEQSEFDVILNNAGASKINVIKVVREITGLG
LKEAKDLVDNAPKPIKEKVGKEEAEAVKAKLEEAGASIEVK
>lcl|NZ_CP026520.1_prot_WP_042235497.1_5742 [gene=rplJ] [locus_tag=PC41400_RS29060] [protein=50S ribosomal protein L10] [protein_id=WP_042235497.1] [location=complement(6408134..6408643)] [gbkey=CDS]
MANAKILAEKEQIVSDLSEKLRASSCTIVTDYRGLNVAQVTELRKQLREAGVEFQVLKNSMVRRATANAELTELDAHLTG
PTAIAFSKDDVVAAAKILNNFAKKNESLSIKAGVVEGKVVDVTQIKALADLPSREGLLSMLLSVLQAPVRNFALAVKAVA
EKQESGAEA
>lcl|NZ_CP026520.1_prot_WP_009674490.1_5743 [gene=rplA] [locus_tag=PC41400_RS29065] [protein=50S ribosomal protein L1] [protein_id=WP_009674490.1] [location=complement(6408915..6409607)] [gbkey=CDS]
MAKHGKKYNEVAKQVTADALYEPAEAIELVKKTAPAKFDETVDIAVRLGVDPRKQDQAVRGVVVLPHGTGKTKRVLVFAK
GDKAKEAEAAGADFVGDQDMINKIQQGWFEFDVCVATPDMMSEVGKLGRLLGGKGLMPNPKAGTVTFDVTKAVQEIKAGK
IEYRLDKAGQIHAPLGKVSFDADKLSENFKALIDALLRAKPAAAKGVYLKSVAISSTMGPSVRVNLQSFR
>lcl|NZ_CP026520.1_prot_WP_009674499.1_5744 [gene=rplK] [locus_tag=PC41400_RS29070] [protein=50S ribosomal protein L11] [protein_id=WP_009674499.1] [location=complement(6409685..6410110)] [gbkey=CDS]
MAKKVIKMVKLQVPAGKANPAPPIGPALGQAGVNIMAFCKEFNAKTADQAGLIIPVVITVFEDRSFTFETKTPPAAVLLR
IAAGVEKGSGEPNKKKVATVKRAKVREIAETKMPDLNAATVEAAMLMVEGTARSMGITIED
>lcl|NZ_CP026520.1_prot_WP_009674510.1_5745 [gene=nusG] [locus_tag=PC41400_RS29075] [protein=transcription termination/antitermination protein NusG] [protein_id=WP_009674510.1] [location=complement(6410290..6410823)] [gbkey=CDS]
MEKRWYVVHTYSGYENKVKANLEKRVESMDMADKIFRVLVPMEEELVNKDGKKKTVMRKVYPGYVLVEMIQTDDSWYVVR
NTPGVTGFVGSTGSGSKPTPLLPEEVESILKHMGMEEPKPKIDFDLKETVRVKVGPFANFVGTVEEIIADKSKLKVHVNM
FGRETPLELEFHQVEKL
>lcl|NZ_CP026520.1_prot_WP_009674493.1_5746 [gene=secE] [locus_tag=PC41400_RS29080] [protein=preprotein translocase subunit SecE] [protein_id=WP_009674493.1] [location=complement(6410846..6411058)] [gbkey=CDS]
MAFLAKMKQGFGSSFSFFGESWSELKKVKWPTRKEMISYTTVTLVTVAFVTVYFFLLDLGISELLRLVFK
>lcl|NZ_CP026520.1_prot_WP_082211310.1_5747 [gene=rpmG] [locus_tag=PC41400_RS29085] [protein=50S ribosomal protein L33] [protein_id=WP_082211310.1] [location=complement(6411077..6411226)] [gbkey=CDS]
MRVVMTLACTNCKQRNYTSNKNKRNHPDRMEFKKFCKYCNEHTAHRETR
>lcl|NZ_CP026520.1_prot_WP_042235494.1_5748 [gene=sigH] [locus_tag=PC41400_RS29090] [protein=RNA polymerase sporulation sigma factor SigH] [protein_id=WP_042235494.1] [location=complement(6411482..6412126)] [gbkey=CDS]
MSVDLKELKMTDYDLKTDEDIVDAVHCGNSDALEYLINKYRNFVRAKARSYFLIGADREDIVQEGMIGLYKSIRDFRGDK
LSSFKAFAELCITRQIITAIKTATRQKHIPLNSYVSLDKPIYDEDSDRTLLDVICGSKVTDPEELIINQEEFSGLEDKMG
EILSDLERRVLMLYLDGRSYQEIAVDLDRHVKSIDNALQRVKRKLERYLEVRDL
>lcl|NZ_CP026520.1_prot_WP_042235490.1_5749 [locus_tag=PC41400_RS29095] [protein=NYN domain-containing protein] [protein_id=WP_042235490.1] [location=complement(6412256..6412768)] [gbkey=CDS]
MKEFLIVDGYNIIGAWPELKQLRDRSLEEARDRLITILAEYQSFSGVKVYLVFDAYQVPGLGKTYVQSRLSVLYTKEKET
ADELIERLVTELTGRRRQIYVATSDMTEQHVIFGKGALRLPASELFVKIKQSRTEVRKRIREEAESGRNTFDSKLNEEMR
EIFERWRRGK
>lcl|NZ_CP026520.1_prot_WP_042235487.1_5750 [gene=rlmB] [locus_tag=PC41400_RS29100] [protein=23S rRNA (guanosine(2251)-2'-O)-methyltransferase RlmB] [protein_id=WP_042235487.1] [location=complement(6412775..6413512)] [gbkey=CDS]
MEEYIGGKHSVLEALRSGRTIHKIWLSESAQKQLAQPVAAEAKKLGVIVQTADKRKLDQLAPDLQHQGVVAQVAAYDYAS
VDDILARAERKGEDPFILILDEIEDPHNLGSILRTADCTGVHGVIIPKRRSVGLTATVSKTSAGAVEYVPVARVTNISQT
IGDLKERGVWVAGTDVTAADDVYRSNFKMPLALVIGNEGRGVGRLIKESCDFLVKLPMHGRVNSLNASVAAGVLMYEVVR
QRTTG
>lcl|NZ_CP026520.1_prot_PC41400_RS29105_5751 [locus_tag=PC41400_RS29105] [protein=ribonuclease III domain-containing protein] [pseudo=true] [partial=3'] [location=complement(<6413734..6414120)] [gbkey=CDS]
MNLTDKLNAQLFRYPPSQKPELLNPLVLAYVGDAVYEMFIRQYVISMPSQRPNHLHRTATRYVSAKAQSKALLALLPLLT
EEETDIVKRGRNAKSGSTAKNADVLEYRHSTAFESLIGYLYYKQAYERL
>lcl|NZ_CP026520.1_prot_WP_042235484.1_5752 [gene=cysS] [locus_tag=PC41400_RS29110] [protein=cysteine--tRNA ligase] [protein_id=WP_042235484.1] [location=complement(6414117..6415514)] [gbkey=CDS]
MTLRMFNTMTREKEEFKPLEPGKVKMYVCGPTVYNYIHIGNGRPAIFFDVVRRYLTYVGYEVTYVVNFTDVDDKLIRKAE
ETGETVQELADRFIAAYMEDMKTLRIEPATIHPRVTDNMEIIITFINSLVQQGFAYENGGDVYYRTSRFKEYGKLSHQNL
DELQYGIRIEVDERKENPQDFVLWKAAKPGEVFWSSPWGNGRPGWHIECSALVHKYLGESIDIHGGGVDLTFPHHECEIA
QTEAITGKPMAKYWLHNEFLNIDNEKMSKSLGNGVLIREIAKKIKPETLRFFMLSSHYRNPLNFSEDGLTQAGNGLDRIQ
NCVTNLKHRLGIAQEDALDDQVEARVKEIEAQFISKMDDDFNTPDAITAVFDLVTEANRVLQQERVSAATLRLIMGRFRN
FDSVLGILAREEDELLDEEIEGLIEERNEARRAKNWARADEIRDLLTEKGIYLEDTPQGIRWRRK
>lcl|NZ_CP026520.1_prot_WP_042235482.1_5753 [gene=cysE] [locus_tag=PC41400_RS29115] [protein=serine O-acetyltransferase] [protein_id=WP_042235482.1] [location=complement(6415495..6416166)] [gbkey=CDS]
MWKTIRSDIKAVFDNDPAARSWFEVVFTYSGLHAIWAHRVAHWLYTRKLFAAARIISQFSRFMTGIEIHPGATIGERLFI
DHGMGVVIGETCEIGNDVILYQGVTLGGTGKEKGKRHPTIGSKVVISSGAKILGSFKVGDNAMIGANAVVLKEVPANSTV
VGIPGKIVKRDGIRVNRLDHGNVPDPVLDLCRTLQVQIDELREQLEEEKIKNGGVREHDTQNV
>lcl|NZ_CP026520.1_prot_WP_042235480.1_5754 [gene=gltX] [locus_tag=PC41400_RS29120] [protein=glutamate--tRNA ligase] [protein_id=WP_042235480.1] [location=complement(6416540..6417997)] [gbkey=CDS]
MSTPLRVRYAPSPTGHLHIGGARTALFDYLLARKNGGQFVVRFEDTDQTRHVESGVQNQLDGLSWLGLNWDESVDIGGPY
APYRQMERLGLYQPFIDQLLSERNAYPCYCTEQELEREREEQEARGETPQYSGRCRRHLTAEQKAEFEAEGRKPSVRFHV
PKDKIIAFKDKVREEVEFETNGIGDFIIVRPDGIPTYNFAVVLDDHLMKISLVIRGEEHLTNTPRQIMLYEALGLPVPDF
AHLSLILNPDRKKMSKRDESIIQFVQQYSELGYLPEAVVNFIALLGWSPGGEVEMFTLEELVEQFDLDRVSKSPAVFDMD
KLNWMNNHYIKQAPLERIAGLAVPHLQKAGLIPAELSEDQQRWATELVGLYQEQLRYAAEIVDLAELFFREKLAFDDEAA
ALLAEEHVPAVVQSFLKQVEAADSLAPENVQKLLKAVQAETGYKGKQLFMSIRVALTGQMHGRDLNMTIHLLGREKVQER
LRALL
>lcl|NZ_CP026520.1_prot_WP_042235478.1_5755 [gene=ispF] [locus_tag=PC41400_RS29125] [protein=2-C-methyl-D-erythritol 2,4-cyclodiphosphate synthase] [protein_id=WP_042235478.1] [location=complement(6418049..6418534)] [gbkey=CDS]
MIRVGQGFDVHQLAEGRPCIIGGVTIPYEKGLLGHSDADVLLHAITDAILGALGLGDIGKHFPDTAAEFKDADSLVLLKR
VWALAKERGYRLGNVDSTIIAQKPKMAPYIPQMAEIIARALDAELEQVNVKATTTEQLGFTGRGEGIAAQSVVCLIKGVL
S
>lcl|NZ_CP026520.1_prot_WP_174446857.1_5756 [gene=ispD] [locus_tag=PC41400_RS29130] [protein=2-C-methyl-D-erythritol 4-phosphate cytidylyltransferase] [protein_id=WP_174446857.1] [location=complement(6418539..6419240)] [gbkey=CDS]
MQVNKVGVIVVAAGRGSRMRTAESKQYLELAGKPILVHTLELFEKMARIDSVVLVTGEADVPRCESYAARYGLHKVSRVI
AGGAERQDSVRSGLEALGAETQWVLVHDGVRPLVTEEQVGACLDKAIDTGAAVLAVPVKDTIKVVSEDGQIMSTPDRKSL
WAIQTPQAFRFSLLKEAHEEAIRDGFLGTDDAMLVERAGHTVHVVEGDYANIKITTPEDLPWAEWLLGNRRGE
>lcl|NZ_CP026520.1_prot_WP_042235474.1_5757 [locus_tag=PC41400_RS29135] [protein=PIN/TRAM domain-containing protein] [protein_id=WP_042235474.1] [location=complement(6419340..6420386)] [gbkey=CDS]
MKWIQWIIAVLGGALGYGWSDAIIQLSGINGGTVADYTAAGAGFIGMFIVSWIGMGLILRLVQRGENSLSQMPASDLLPA
AFGTIVGLLAAVLVFPALERVHVVLAGMATVLLAFMGGRIGYGKSEEIFELLSSRKDGGSGKGKMPEEHKILDTSVIIDG
RIADICKTGFIEGTLVIPEFVLEELQHIADSSDLLKRNRGRRGLDILNKIQKELDVKVLIYEGDFEEISEVDSKLIRLAK
VLQGKVITNDYNLNKVCELQGVSVLNINDLANAVKPVVLPGEEILVQVIKDGKEHGQGVAYLDDGTMIVVEGGRDFIGSL
LEVMVTSVLQTSAGRMIFAKPKLLEKAQ
>lcl|NZ_CP026520.1_prot_WP_042235473.1_5758 [locus_tag=PC41400_RS29140] [protein=hypothetical protein] [protein_id=WP_042235473.1] [location=6420549..6420944] [gbkey=CDS]
MSALSLKDFQEQVGELLLRHRSLLDVLSKFQQSNAGVNRSVIKSITECGCVEVNAKKQEYSPEMTLDQAKIELQTHLEGQ
LCEHCMEAISTEIGKHLFYLSALSNLLDVNLEKVLEDESKKCSTLGFFNMS
>lcl|NZ_CP026520.1_prot_WP_042235471.1_5759 [gene=pssA] [locus_tag=PC41400_RS29145] [protein=CDP-diacylglycerol--serine O-phosphatidyltransferase] [protein_id=WP_042235471.1] [location=complement(6421069..6421815)] [gbkey=CDS]
MITKSLPSLFTVGNLFLGVLAIILVFNERPDTAAVLVIVAMLLDGLDGRVARALNAQSEFGKELDSLSDVISFGVAPAFI
MYVVAFGDIRPAAAAWVVTAIFPICGALRLARFNVVAGTPGYFIGLPIPAAGGILATLALFHNELNVYVLLVSTVLLSYL
MVSNVKYPNFKKAGIPKAAIWITPIIVIIAVIIGINKPSFISKMIFIPLVIYALYGLKKNVDRLFVLRRRKHKAMETTED
LANSKTHL
>lcl|NZ_CP026520.1_prot_WP_042235469.1_5760 [gene=disA] [locus_tag=PC41400_RS29150] [protein=DNA integrity scanning diadenylate cyclase DisA] [protein_id=WP_042235469.1] [location=complement(6421902..6422981)] [gbkey=CDS]
MSKEETKRDVMSQLLQLVAPGTPFRDGLENVLRAKTGALIVVGCTPEVMEIVDGGFSIDCDFTPNYLYELAKMDGAIILN
EDAKKILYANTQLIPNSSIPSTETGIRHRTAERVAKQTNKLVVSISQRRNIITLYQGNLRYSLKDIGVILTKANQAIQTM
ERYKTELDQSLINLGASEFEELVTLHDVTNAIQRVEMVLRIKNELNRYITELGTEGRLISMQLEELVGNTELEVRLLLKD
YVRELSEEKIKDILLGLKRLSSEELLDPHHIIRLLGYPHTNAVSEELVSPRGFRVLNKIPRLPSIIISNLVEKFGHLPHI
TVATIEELDEVDGIGEVRARAIKEGLRRIQEQVLIDRHI
>lcl|NZ_CP026520.1_prot_WP_042235467.1_5761 [gene=radA] [locus_tag=PC41400_RS29155] [protein=DNA repair protein RadA] [protein_id=WP_042235467.1] [location=complement(6423095..6424462)] [gbkey=CDS]
MSKTKTKFICQQCGVETPKWLGKCPGCGAWNSFAEEVETVVKTQGMNSSFLRTKEKPTPIINIESSVEQRVLTDNGELNR
VLGGGFVPGSLILVGGDPGIGKSTLLLQTSHSLTKEGRKVLYISGEESIRQTKLRADRLDAVSPNLYVLCETNMEVVEEA
IQEVQPDFLVIDSIQTVYHPSVSSAPGTVSQVRECTSHFMRIAKLKNIATVLVGHVTKEGAIAGPRLLEHMVDCVLYFEG
ERHHTYRVLRAVKNRFGSTNEIGIFEMRETGLAEVSNPSELFLSERPLGVAGSTVVASMEGTRPVLVEIQALVSSTNFPS
PRRMATGIDHNRLSLIMAVLEKRVGLFLQSQDAYLNVAGGVKLDEPAVDLAVAISIASSFKEQPTQAYDAVFGEVGLTGE
VRGVSRIDQRVKEAQKLGFRRVIMPEKSLKGWTPPKGIEVVGVSTVSEALKAALD
>lcl|NZ_CP026520.1_prot_WP_042235464.1_5762 [gene=clpC] [locus_tag=PC41400_RS29160] [protein=ATP-dependent protease ATP-binding subunit ClpC] [protein_id=WP_042235464.1] [location=complement(6424620..6427067)] [gbkey=CDS]
MMFGRFTERAQKVLSLAQEEAVRLGHNNIGTEHILLGLIREGEGIAAKALVALGLGLEKIQDEVESLIGRGQEQPSNIAY
TPRAKKVIELSMDEARKLGHTYVGTEHILLGLIREGEGVAARVLNNLGISLNKARQQVLQLLGSSEVVSSNHGSGNNVNT
PTLDSLARDLTAIAKEGNLDPVIGRSKEIERVIQVLSRRTKNNPVLIGEPGVGKTAIAEGLAQKIINNEIPETLKDKRVM
TLDMGSVVAGTKYRGEFEDRLKKIMDEIRQAGNIILFIDELHTLIGAGGAEGAIDASNILKPALARGELQCIGATTLDEY
RKYIEKDAALERRFQPITVDQPTPEEAIQILHGLRDRYEAHHRVKITDEAIEEAVKLSDRYIPDRFLPDKAIDLIDEASS
KVRLHSYTVPPSLKELEQKLENIRKEKDAAVQSQEFEKAASLRDTEQKLREELDSTKNEWKEKQGRMDTEVTPDDIAQVV
ASWTGIPVRKLAEEETERLLKMEDILHNRVIGQEEAVKAVSRAIRRARAGLKDPKRPMGSFIFLGPTGVGKTELARALAE
SLFGDENAVIRIDMSEYMEKHSTSRLVGAPPGYVGYEEGGQLTEKVRRKPYSVVLLDEIEKAHPEVFNILLQVLEDGRLT
DSKGRTVDFRNTLIIMTSNVGADTIKKNSTLGFTAVQDSGKDYNNMKDKVMNELKKSFRPEFLNRIDEIIVFHSLDQKHI
AEIVSLMADELRKRLKEQEVDFTLTDDAKNFLAKEGFDPSYGARPLRRAIQKHIEDRLSEELLKGSISKGDSLTIDEKDG
ELVVLRNEEAMTQTK
>lcl|NZ_CP026520.1_prot_WP_241688924.1_5763 [locus_tag=PC41400_RS29165] [protein=protein arginine kinase] [protein_id=WP_241688924.1] [location=complement(6427114..6428142)] [gbkey=CDS]
MKGDGPDSDIVISSRIRIARNLQDYPFPMLAANQQSAEVLEKVSDVLGNEDLKTISDFTLVPLSELNELERRILVEKHLI
SPNLATESRNGAVILNDNESISIMVNEEDHLRIQCLCPGFQIKEAWDLANQIDDIFESQLDYGFDERRGYLTSCPTNVGT
GIRASVMMHLPALVLTQQINRILSAVSQVGLTVRGLYGEGSEALGNLFQISNQITLGQSEDEIIDNLYSVAKQIIEHERA
ARRQLMQESRIRITDRVMRSYGILTHASIIDSKEAAQRLSDVRLGIDLNLVGEVPAHTMNELLVMTQPGFLQQIAGELLS
PDDRDIRRAEIIRERFKPCHNV
>lcl|NZ_CP026520.1_prot_WP_042235459.1_5764 [locus_tag=PC41400_RS29170] [protein=UvrB/UvrC motif-containing protein] [protein_id=WP_042235459.1] [location=complement(6428230..6428748)] [gbkey=CDS]
MLCQECGKRPATLHFTKIVNGEKTEFHFCETCAKEKGELIPGTPGGFSIHNLLSGLLDFDPSSGTVSSKARPIRCEHCGL
TYSQFSKLGRFGCSHCYESFQDKLDPLFKRVHGNTQHVGKVPKRSGQEIEQKREVERLKKELQTYIEQEEFEQAAELRDR
IRDLDKKSAGSS
>lcl|NZ_CP026520.1_prot_WP_009674502.1_5765 [locus_tag=PC41400_RS29175] [protein=CtsR family transcriptional regulator] [protein_id=WP_009674502.1] [location=complement(6428761..6429222)] [gbkey=CDS]
MRNVSEIIEQYLKHILQQSPEGSIEIQRNELADQFQCVPSQINYVISTRFTLEKGYVVESKRGGGGYIRIQKIELNAYGS
LLDTIMTTIRESIDQASAEGLIYRLEEGRLISRREAAIIKAAISRDVLQFKLPLRDEVRARILKAMLISLLSK
>lcl|NZ_CP026520.1_prot_WP_042235226.1_5766 [locus_tag=PC41400_RS29205] [protein=pro-sigmaK processing inhibitor BofA family protein] [protein_id=WP_042235226.1] [location=complement(6435093..6435356)] [gbkey=CDS]
MKTAVWGMLALSAGALLVLLVRQPGARKIFSSIGVHVVVAAFLLYGVQLLSGYSRFELPINIYTVGTVSVLGVPGLMLLA
ALKVVLV
>lcl|NZ_CP026520.1_prot_WP_042235223.1_5767 [locus_tag=PC41400_RS29210] [protein=DUF2508 family protein] [protein_id=WP_042235223.1] [location=complement(6435353..6435628)] [gbkey=CDS]
MNDLRKSAFSFQRGTRKTAEQDKQELLAEIRLAHSQWKTAQHHFEHALEKDEIDYAIYAVEAAEKRYEMLLRQAKKLNVT
SAYHITAEVRG
>lcl|NZ_CP026520.1_prot_WP_042235221.1_5768 [gene=recR] [locus_tag=PC41400_RS29215] [protein=recombination mediator RecR] [protein_id=WP_042235221.1] [location=complement(6435798..6436394)] [gbkey=CDS]
MHYPEPISKLIDAFTRLPGVGPKTAGRLAFHVLRMKEEDVLDFAKALVNVKRNLHYCSVCCNITDIDPCRICQDKSRDAS
VICVVQEPKDLVAMERTREFDGYYHVLHGAISPMEGIGPDQIHIAELLKRLGDEQVKEMILATNPNIEGEATAMYLSRLV
KGFGLKVTRIAHGLPVGGDLEYADEVTLTKALEGRREI
>lcl|NZ_CP026520.1_prot_WP_009671602.1_5769 [locus_tag=PC41400_RS29220] [protein=YbaB/EbfC family nucleoid-associated protein] [protein_id=WP_009671602.1] [location=complement(6436457..6436768)] [gbkey=CDS]
MNNMNQMMKQVKKMQEQMLKVQEELGSKTVEGTAGGGVVTVTANGHKKVTAIAIKPEAVDPDDVEMLQDLVLTAVNDALN
KAEELAGKDMGKFTGGMNIPGLF
>lcl|NZ_CP026520.1_prot_WP_042235218.1_5770 [gene=dnaX] [locus_tag=PC41400_RS29225] [protein=DNA polymerase III subunit gamma/tau] [protein_id=WP_042235218.1] [location=complement(6436804..6438543)] [gbkey=CDS]
MTHIALYRTWRSQAFRELVGQKPIVQTLQNSLREQRFTHAYLFSGPRGTGKTSAAKILAKAVNCVHGPAEEPCNECEACI
RISQGSVMDVVEIDAASNRGVEEIRDIRDKVKYAPTEVRYKVYIIDEVHMLTTEAFNALLKTLEEPPAHVIFILATTEPH
KLPATIISRCQRFDFRRVSMEEQVGRLAYVCREENIAIEEEALHYIARLSEGGMRDALSLLDQASSFAVGTIAASDVVAM
TGGLASDQFEKLVGSMKEQDIGAALELIDQFMQEGKSADKCMENLIHYFRDLLMIRLVPDSSVLTERVFDVARLDQVASR
FGTDQLIAIIEVLNQYQSEMKYSVQPQTILEVAVMKACGAIASGGGSRPAEAPPASRSAGEPEPEALSQLTRRIQQLEEQ
LAKLMQGGVPAAGAQQPARSPVAPRSSNAAVRKASVRLDDFTSAVESHHTRALLGKWTQILAMVKERKITVHAWLVDGEP
VAIHGDTVLVAFKSAMHRDTTEKPANKQLIEQVISEALGETLKLATVMQKEWKEAREGASEPAREEMTLQAEDSEPDKEP
WVDEAIKLFGEDLVSIKDE
>lcl|NZ_CP026520.1_prot_WP_241688925.1_5771 [locus_tag=PC41400_RS29230] [protein=ATP-binding protein] [protein_id=WP_241688925.1] [location=complement(6438630..6439961)] [gbkey=CDS]
MYANEPFLKRTGYTAEDMKMAPFSEVIQLPEPVDCFYSRHFDMGSDKTDAKYPIAIVKDSQGFIQLQKCSMLFAEVNSEP
HYLFLLDDLEEVSEQNLLQRFSGTFLRDVNLGVLLIDTEFRLVDISDMACRVLGFHKESILFKRIEDIFASVPPEHRLVE
RSLLDGRIVRNHAVSWMNNQERYELLLDSNLLRDNEDQIVGAYIIFKDVTNLRSLEAQVQRSDRLAMIGQIAAGTAHEIR
NPLTSIKGFLQVLKKTFESKDMAKEQGYIDVMLDEINRVNELVNEFLLLGKPKNVLLQKIDVSTVIKGILPIINNEAILH
KVHMNYEESSGLPPVIADQELLKQVFLNICKNGIEALTDGGTLTIVERLDPDGRFVNIEVHDTGPGIPVYVIDKIFDPFF
TTKEEGTGLGLSVCQRIIHDMGGHIRVSSKGYGTTFTVSIPCT
>lcl|NZ_CP026520.1_prot_WP_042235213.1_5772 [locus_tag=PC41400_RS29240] [protein=hypothetical protein] [protein_id=WP_042235213.1] [location=complement(6440626..6441027)] [gbkey=CDS]
MGNVIVKRSAGGLLAALVTAAAITGCSPGQPAGPKAYNYSSDGYLGTTSANPNLALNPGYHTYQVDSDIALSVLGRLKGV
VDTQLQFKGPIGYVKLSVPRGTPPEKAAAIRAQAEQDLHTQLPRYRFIVTVKR
>lcl|NZ_CP026520.1_prot_WP_016423669.1_5773 [gene=rpmE] [locus_tag=PC41400_RS29245] [protein=50S ribosomal protein L31] [protein_id=WP_016423669.1] [location=complement(6441128..6441325)] [gbkey=CDS]
MKQGIHPNYHETTVTCACGNTFESGSVRENLRVEICSNCHPFFTGKQKFIDAGGRVDKFKKKYGI
>lcl|NZ_CP026520.1_prot_WP_042235209.1_5774 [locus_tag=PC41400_RS29250] [protein=radical SAM protein] [protein_id=WP_042235209.1] [location=complement(6441430..6442692)] [gbkey=CDS]
MNVVYADPSGNVFDHPDYEALGRSADQIVELLEEELIPLPEGATLVSLPHTRAVGINTETGEMEVLPGEYAAVGALLPQG
FTRLMLPGYVKTDKEEKFPLFGYTAVVWKDGAFYVAAEQCDDPEPWNPRNCDPDELEVSVEKLRARYPENRLYEHLSKCA
LEYECLTASNTFLNRWEGAVPVSFSCNAGCFGCISEQPDDSGFPAPQTRMNFKPQAKELAEVMLEHLKTPDSIISFGQGC
EGEPSTQAKLIIEAMREVRSRTDMGYININTNAGLSDHIRGIVDAGLDLMRVSTISALDDHYNAYYKPRGYTLANVEKSL
KYATSKGVITSINYLIFPGVTDREEEVEAMIEFVRRTGLRLIQMRNLNIDPESYLNLIPKAQGDILGMKQMLEIYREELP
DVVIGSYTHIPAFFDRAQRA
>lcl|NZ_CP026520.1_prot_WP_009671576.1_5775 [gene=rho] [locus_tag=PC41400_RS29255] [protein=transcription termination factor Rho] [protein_id=WP_009671576.1] [location=complement(6442747..6444039)] [gbkey=CDS]
MDMHLAQLEALKLTELYKLAKKYQIPYYGQLKKKELIFSILRAQAAESGLMFMEGVLEVLQEGFGFLRPINYLPSTEDIY
ISASQIRKFDLRSGDLVSGKCRPPKENERYFGLLHVEAVNGESPDTASERLHFPALTPLYPQTKVVLETSPEKLSTRIMD
LMAPVGLGQRGLIVAPPKAGKTLLLKEIANSISTNYPEIELFVLLIDERPEEVTDMQRSVKGEVVASTFDELPENHIKVA
ELVLERAQRLVEHKKDVVILLDSITRLARAYNLVVPPSGRTLSGGIDPGAFHRPKRFFGAARNIEEGGSLTILATALVET
GSRMDDVIYEEFKGTGNMELHLDRKLAERRTFPAIDIRRSGTRREESLLTKEELEKVWGLRKSMNDTYEYTEAFIKKLGE
TKTNQEFLDSLETNNGNVTLGKGKMKNSAS
>lcl|NZ_CP026520.1_prot_WP_042235207.1_5776 [locus_tag=PC41400_RS29260] [protein=UDP-N-acetylglucosamine 1-carboxyvinyltransferase] [protein_id=WP_042235207.1] [location=complement(6444133..6445392)] [gbkey=CDS]
MEKLMIAGGRPLKGTVTISGAKNSAVALIPAAILAESTVTLDNLPHLSDVAIYTEILSELGASIEWNNEQMIIDPSAMKS
KPMPNGNVKKLRASYYLMGAMLGRFGEAIIGLPGGCNFEPRPIDLHIKGFEALGAEISNENGSLHIRAKQLRGAKIYLDM
VSVGATINIMLAASRAKGFTLIENAAKEPEIIDVATLLNSMGAKIKGAGTETIRIEGVDEMHGCRHSIIPDRIQAGTYMI
AAAATRGDVTVDNIIPKHMEAMTAKLQEMGVEIHEMDESIRVVGRALYDSIDVKALVYPGFATDLQSPMTSLLTQTTGVS
MLTDLVYSNRFKHVPELVRMGAKIKVEGRTAIIEGGTLSAAKVKASDLRAGAALVIAGLTVPSDVTEVTGVEYIDRGYDN
LVANLTLLGAEVWRENGEE
>lcl|NZ_CP026520.1_prot_WP_009671622.1_5777 [gene=fba] [locus_tag=PC41400_RS29265] [protein=class II fructose-1,6-bisphosphate aldolase] [protein_id=WP_009671622.1] [location=complement(6445712..6446566)] [gbkey=CDS]
MPLVSMNEFLPKAKANKFAVGQFNMNNLEFAQAITEAAMELNSPFIFGVSEGALKYMGIEYTVAIAEAAAKQSGLPIALH
LDHGSSFEVAMKCIRAGFSSVMFDGSHYSYEENVRLTKEVVKAAHAMGVSVEGELGTIGGVEDDISVDEADANLAKPEEA
IRYYEETGVDCVAIAVGTAHGMYAGEPNIHFDIIEKVSQAIPIPVVLHGGSGVPDEMIKKAIAAGAGKVNVNTENQVAAT
AAIREVLGADAKVYDPRKYLTPARKAMVEVVKSKIQLFGSANQA
>lcl|NZ_CP026520.1_prot_WP_042235205.1_5778 [locus_tag=PC41400_RS29270] [protein=response regulator] [protein_id=WP_042235205.1] [location=complement(6446724..6447104)] [gbkey=CDS]
MSKKFLIVDDQNGIRMLLMEVFSNEGYETYQAPNGKTALEIVKTHSPDLVLLDMKIPGMDGLDILKHVKSIDPSIKVIMM
TAYGELDMIKEATELGAIMHFTKPFDIDELRLAVHTQLIETTPLGS
>lcl|NZ_CP026520.1_prot_WP_009671614.1_5779 [locus_tag=PC41400_RS29275] [protein=CTP synthase] [protein_id=WP_009671614.1] [location=complement(6447265..6448881)] [gbkey=CDS]
MTKYIFVTGGVVSSLGKGITAASLGRLLKNRGLKVTIQKFDPYINIDPGTMSPYQHGEVFVTDDGAETDLDLGHYERFID
INLSKNSNVTTGKVYSSVITKERRGEYLGGTVQVIPHITNEIKDRVFRAGREAGSDVVITEIGGTVGDIESLPFLEAIRQ
IKSDIGRENVMYIHVTLIPYIKAAGEVKTKPTQHSVKELRSLGIQPHVIVCRTEHALTEDMKRKIAQFCDTDPEAVVECL
DAETLYEVPMMLREQGLDDIVVNHLKLGGSAQPDMTEWEQLVQRVKGLKETTEIAIVGKYVALHDAYLSIVESLGHAGID
NGAEVKIRWVNAEEVYDHNVDELLGGVDGILVPGGFGDRGIEGKVIAIRHAREKKIPFFGICLGMQVAVIEYARALANLE
GANSSEINPVTPYPVIDLLPEQKDIENLGGTMRLGLYPCKLAEGSLAMQCYNDELVYERHRHRYEFNNEYREMIESAGLR
ISGTSPDGRLVEIVELPDHPWFLAVQFHPEFTSRPNRPQPLFREFVKAAFTYKQESRA
>lcl|NZ_CP026520.1_prot_WP_042235203.1_5780 [gene=rpoE] [locus_tag=PC41400_RS29280] [protein=DNA-directed RNA polymerase subunit delta] [protein_id=WP_042235203.1] [location=complement(6449181..6449768)] [gbkey=CDS]
MSSPYTLKIKPEKAAEMPMVDLAFEVLKAANTPFYYRDLMQEIAKIKGLAEDETMNVIAQLYTEINIDGRFACVGTNLWG
LKRWYPVEKSDDAVGNAKRPRIINDEDDDLDEDPYSEEEEYKEEEYDTFTADGFDKNDAEEEEEEEEEEEESEFFEEEEL
DEELEDAELDEEVDEESEDDSDDDEDFEADDEKDK
>lcl|NZ_CP026520.1_prot_WP_042235200.1_5781 [locus_tag=PC41400_RS29285] [protein=S8 family peptidase] [protein_id=WP_042235200.1] [location=6450139..6451317] [gbkey=CDS]
MDSKRLLMMLHEEIQASQGERRHIILFQSDRDYRQVTELIRLHKDTHSSLSSVYDLRLIRGISCPIRNRAELIKHPAVLS
IEEDTSINIQLRPRKTVKTSEIVEKTSEGSQFIPWGIQQIKAPDVWKNAKGSRITIGVIDTGIDYYHPDLQASVARGINL
LNRSQPPYDDNGHGTHIAGTIAAAGRLGITGVAPKAVIHPVKAFDMNGSAFVSDIVAGIDWCVRSGLDIINMSFGMKTYS
RALEEAVINAYQSGTIVVASSGNDGKGAAIDYPARFPSVFSVGATTKVNKVAPFSNRGKRIDIYAPGESIYSCWLHGKYN
ELSGTSMATAHVSGVIALMLSVRPSLRPQQIKQILRRSATPLTKAKQPEGTEAPGLIHARRAITALLKTNKT
>lcl|NZ_CP026520.1_prot_WP_042235198.1_5782 [gene=argS] [locus_tag=PC41400_RS29290] [protein=arginine--tRNA ligase] [protein_id=WP_042235198.1] [location=complement(6451495..6453168)] [gbkey=CDS]
MNVLEQVKETLKVTIGEAAVAAGLVQASELPEIVLEVPKDKAHGDFATNLAMQLSRIAKQNPRAIAEQLIQKLDKEKAQI
AEVQIAGPGFINFRMDKSYLYPILAQVAEQADRYGAVNVGQGRRVQMEFVSANPTGSLHLGHARGAAVGDALCNVLALAG
YNVTREYYINDAGNQVNNLAKSIEARYLQTLGQDAEMPEDGYHGEDIKGFAGELAAEEGDRLTKLGQDERLAYFREYGLK
KELDKIKRDLERFGVKFDEWFSETSIYENNLIDDVLGKLRESGYVYELDGATWLRTTELGDDKDRVLIKNDGSFTYLTPD
VAYHRNKFQRGYDELINIWGADHHGYIPRMKAAMTALGYDANRLTVLIAQMVSLFQNGEKVKMSKRTGKAVTMEDLMDEV
GVDAIRYFFTMRSMDSHLDFDMDLAISKSNENPVFYVQYAHARICSIFRQAEEQGVELRGLAQVDLSQLSTEAEYELLRK
IGELPQEVAVAAELYAPHRLIRYVYDLAGLFHSYYRAERVITEDAEQTQARLALLGALRTVIANVLAVIGVSAPDRM
>lcl|NZ_CP026520.1_prot_WP_042235196.1_5783 [locus_tag=PC41400_RS29295] [protein=DUF1934 domain-containing protein] [protein_id=WP_042235196.1] [location=complement(6453165..6453581)] [gbkey=CDS]
MSNKKNVRIRITSVAGQESIIQEARGELYIREGKVYVLYQETSEEMGSTSTMVKAGPEELKITRKGQIGSEQTFVAGHSR
TGWYRMPEGSIPIETRTRTYVRNLQEGIGVIDWSYDLYVGEERAGLYTLSLTIQEDVR
>lcl|NZ_CP026520.1_prot_WP_042235193.1_5784 [gene=speB] [locus_tag=PC41400_RS29300] [protein=agmatinase] [protein_id=WP_042235193.1] [location=complement(6453855..6454724)] [gbkey=CDS]
MRLDQAYSGNIFIAASEDYASSRAVLYGMPMDYTVSFRPGSRFGPARIREVSIGLEEYSPYLDKSLDEVKYFDAGDLMLP
FGNAGRSLDVIGEYVRGLLSDGKFPLGMGGEHLVSWPVIREVYAKYPDLALIHIDAHADLREQYEGEPLSHSTPIRKAAE
LIGGKNVYQFGIRSGMREEFEYARKNLNLYPFEVLEPLKKVLPELAGRPVYLTIDIDVLDPSAAPGTGTAEAGGITSKEL
LAAVHAIAGSELNIVGADVVEVAPVYDPSEQTQIVAAKLIREILLGLVK
>lcl|NZ_CP026520.1_prot_WP_042235191.1_5785 [gene=speE] [locus_tag=PC41400_RS29305] [protein=polyamine aminopropyltransferase] [protein_id=WP_042235191.1] [location=complement(6454759..6455586)] [gbkey=CDS]
MELWYTEKQTEDFGITAKIKETLVTEKTDFQDLAMIDTAEWGRMLVLDGMVMTTVRDEFVYHEMVAHPALVTHPNPKNVL
VVGGGDGGVIREVLKHPKVEKAVLVEIDGKVIEYSKKYLPEIAGELDNPRVEVIVNDGYMHIHDHKNTYDVIMVDSTEPV
GPAVELFTKGFYQGIYDALKEDGIFVAQTDNPWFKADLIQTVNRDVREIFPIVRVYGANIPTYPSGLWTFTMGSKKYDPL
QVEEASIPEFETKYYTPRLHKAAFALPKFVEDLVK
>lcl|NZ_CP026520.1_prot_WP_042235189.1_5786 [locus_tag=PC41400_RS29310] [protein=PBP1A family penicillin-binding protein] [protein_id=WP_042235189.1] [location=6455962..6458040] [gbkey=CDS]
MADDQQNEPENNAAPDKKPPGPADKNWPRRLKRGFTFSFTTFLVLCVLGGVFLLYLRSQALPVTKVLQTSQIFDVNGALI
DTLDSGQNRKIVPISDISVYVVKATLAIEDQNFYSHPGFDVKSIARAAIVNLKSMSKVQGAGTITQQLARNLYLNHDRTW
TRKIKETVYAVQMEMQLTKDQILESYLNQIYYGHSTYGIEAASQLFFGKHAKDLSLAQSAMLVGVPKGPKFYSPYYNMEN
AMDRKNVVLQTMAKEGFITQQEADAAKQEKLTVQPLVEKKPSQAPYFRDYIKTLAVQKLGITEDQFEQGGLNIYTTLDLK
AQKIAEEQMAKQLPKESEIQGALISLDPRTGYVKAMVGGKDYGANQFNRVFATTRQPGSSFKPFVYLTALQNGFTPLSTV
RSEPTVFKYDEGRQSYTPRNYNDQYENADIDMRRAIAKSDNIFAVDTILKVGPQKVIDNARKFGITGDLKPLPSLALGTY
PVSPYEMVAAFSALANQGVRVEPTAIIKIEDMAGSVLYQAAPKSERVADAAPAYVLTHMMESVFEEGGTGSRVAATIKRP
VAGKTGTTDTDAWMVGYTPELATAVWVGYDKGRAINTAESHLASPIFAGYTEAVLEAIPPKLFPVPEGVTTVYIDPKSGK
LANEDCTTGARNEAFLSGTEPKEYCTNRAAKGKGFGSGARSWWEDMKRWWNE
>lcl|NZ_CP026520.1_prot_WP_042235230.1_5787 [locus_tag=PC41400_RS29315] [protein=YwhD family protein] [protein_id=WP_042235230.1] [location=complement(6458505..6459011)] [gbkey=CDS]
MDSHKEEPKKKPAALNIVSTKVNHKGFGAGSIDLSNVSSLILDGDEAYLDMGALHAKSQVEKRIKFSTNKEDVPDGRRCW
IVWVAVDRSPEGTSYGGVTACEMLIDTEARKGWKILADHVNRMDAAMKRKIMVEGLDETEKSALRKALIEHNEEWWNHSS
EELKQALA
>lcl|NZ_CP026520.1_prot_WP_241688756.1_5788 [locus_tag=PC41400_RS29320] [protein=M1 family metallopeptidase] [protein_id=WP_241688756.1] [location=6459130..6461157] [gbkey=CDS]
MKVKMANRKKAWLVSTAAAFLGAAFLANSLLSSPEERTGQTSRAVLSGTAQNPVPPVNAEPSSSIGQPPATGEGVHQPLS
KRIVEYHINVEYSPENQKLLGQQTVTWKNPGSQPVQELYMHLYPNAFASEKTTFMKESGGKLRRDEAKKGSFGDMELRSI
KTEDETDLSSAMQFVQPDDGNPDDRTLIKITLPKPVAAGAAITLHTDFAVKLPQVFARMGITKDFVMAGQWFPKLAVYEP
AGTRGRTAEGWNLHQYHGNSEFYADFGIFDVRVKVPADYIVAATGFPTKNPVVEAGKKTYQFYADDVHDFAWSASPNFKV
FEEPYAAPNLPGVKIKLYLDPKHEPYKSRYMIAAKKALSRYSQWYGTYPYSTLSIVVPPEEANGAGGMEYPTLVTAWGAS
EGTPDLDLERVVVHEIGHQFWYGLVATNEFEEAWLDEGFTSYTEDKLMELEYGSKPNLPVEASYITSPAALQKNAWSYRS
HSEYADNVYTRAKLVLKEIEHRVGLNTMDRIMKTYFQRWKFKHPGTRDFQTVVEEITKTKWTEFFDQYVYGSMMTDYAVD
NIRIRQTNESGQSLYESSVLVRKFGGTVPAVPIRFHFADGSTTEKVWDGQGEQIEYKLTHKSPVDWVSLDPKYTMKLENR
HINNYMKTEVDTTWRIRWTLGVVKLLETLFDTVAW
>lcl|NZ_CP026520.1_prot_WP_042235186.1_5789 [locus_tag=PC41400_RS29325] [protein=hypothetical protein] [protein_id=WP_042235186.1] [location=6461167..6461988] [gbkey=CDS]
MLRYLRQGLTAALKQPFAVCVLYVYQFAWGILLYKWVQSIIVPLLHRYPGSDQPQSAVHLFLAESQFRLVKTDLIQPYLE
WLLILLAVRMLISPMLNAGVLYSLEHRQMNAGYRFMKGIRELLLPFFLYYLLQTVLMLAPLYWLAPKALALFNKAYSFEP
LLLSALPWAALYAVYALLIQLLFLFLQFGRASEDRRGLLAVLPVVFRQGLRITGNALLLVLLGFLLAVAATYASYIWAGL
TALIIHQIYPLVRTFVKMWSIAAQREIWESYRI
>lcl|NZ_CP026520.1_prot_WP_009671606.1_5790 [locus_tag=PC41400_RS29330] [protein=SurA N-terminal domain-containing protein] [protein_id=WP_009671606.1] [location=6462193..6462783] [gbkey=CDS]
MISPRTSTVVAAVIGLCVAAGSFFYGQSDVKAKEVVAIGHDVRISNQAFDKMKAEVDLAFQMQGAQNNLTNDQLLDLMLK
DELFVRYGQKRGVKVKEAEVKQAIDQQRKALEAAGPEAAQLKEMLYKSAGLTEATYWTDPKTVNQYAKYLLQQKTIEYLV
KKGELKTQEDLNALQEKLLAETKPGLRVKFPDQPKT
>lcl|NZ_CP026520.1_prot_WP_009671580.1_5791 [locus_tag=PC41400_RS29335] [protein=C40 family peptidase] [protein_id=WP_009671580.1] [location=6463110..6463574] [gbkey=CDS]
MKKVLLFVLSLTLFLSVEIGSASAKSTLSSTIDGLVGTPYKYSGTSTSGFDCSGFTSYVFDRFGIDLARTSKAQNGQGSS
VAKSDLRPGDLVFFNTDGKGISHVGIYIGNNKFAHSATNKGVTKSDLSESYYAKRYVSARRVIGDDLFNKITAD
>lcl|NZ_CP026520.1_prot_WP_042235228.1_5792 [locus_tag=PC41400_RS29340] [protein=deoxyribodipyrimidine photo-lyase] [protein_id=WP_042235228.1] [location=6463725..6465116] [gbkey=CDS]
MILFIHRKDLRIDDLTAFDYIHSLGRPSLHVLILDPFLLRDGRDREHSGINFLRHVSRLIREYRKAGAHLEVLAGEPHVV
VDQLLDALPIEEIVLNGDITPYALVRDRKLKVSASARGVPVTTLIDHTLADFGDFQHFAERRTPYQVFTPFYKKWNAYMH
LFFRPSSAVTVRDLQTRPLPDREAWSARFPLPLDPEAAGTAPDPAQTLSDFLEGGLRAYEAQRDRYAFETTSGISRHLNT
GALSVRRAYEAALGSERHEAWTRQLAWRDFYLYQAVYNPDFFVYEKKFDFSGLEDTLFERWCRGQTGIPVVDAAMAELNA
TGHMPNRLRMVTAMFLTKNLRCPFTLGERYFRRKLADYDNVLNRGGWLWSASLGFDASPYFRVMNPVTQSETHDPTGMYI
RRWLPHLAHLGDKEIHKPQADAIVDLKSSRAAAIEIYKHMLAEAKPDAGAGPAGFSPDGFPPV
>lcl|NZ_CP026520.1_prot_WP_009671636.1_5793 [locus_tag=PC41400_RS29345] [protein=DUF1540 domain-containing protein] [protein_id=WP_009671636.1] [location=6465158..6465379] [gbkey=CDS]
MENPMVKCSIANCEYWEQGNNCSAEVIMIEVNAHSNKPQPQERDGLPYDSGHQDSAEGITDTCCQTFEERHHH
>lcl|NZ_CP026520.1_prot_WP_042235183.1_5794 [locus_tag=PC41400_RS29350] [protein=SRPBCC domain-containing protein] [protein_id=WP_042235183.1] [location=complement(6465586..6466035)] [gbkey=CDS]
MDLSYHFYIDGSPEQVWEVFVTPEKTRKIFFDCVIDSSYVPGESIRFIGPGNDGEQTVHIYGEVLEYEPNRILSYIEHPG
PTYYPNHADITTRVTIKLEPFAGCTKLTLINDQWVDNHPSYEKADPAWWMLLSAIKTFVETGKKLDLGW
>lcl|NZ_CP026520.1_prot_WP_042235181.1_5795 [locus_tag=PC41400_RS29355] [protein=helix-turn-helix domain-containing protein] [protein_id=WP_042235181.1] [location=6466181..6466993] [gbkey=CDS]
MKPEENPAVVNKGILNYEEGRQRFTISRRLPAPELAFFIRHYWLIHWDLRGLPSHTQEVLQPPCFNLVFEKDNTRVYGTN
TGKMSQLLEGKGEVFGVLFRPGGFYPFYRQPASELVNRSLAFRKVFGISSLPLDDALFSHSDEDTKVRYVDDFFKKERPS
KDKQIETVNSVIDYIAEHREINRVEDVGVKFGISVRTMQRMFSQYVGVSPKWVIKRYRLQDAAVSADAGESPDWSKLAVD
LGYYDQAHLIKDFKTVIGKSPEEYVRSKQG
>lcl|NZ_CP026520.1_prot_WP_042235179.1_5796 [locus_tag=PC41400_RS29360] [protein=MFS transporter] [protein_id=WP_042235179.1] [location=complement(6467094..6468359)] [gbkey=CDS]
MSSLEARIQAERMPGPASSLFRERAFAMLWLSTIFSGLALNMFYFAEAWYVVNVLNMEASLGVIYMIGTLPRVLFMVIGG
VLADRISPSKIMFVSTITKSALLAVLVLVLVEGGGIGLGALTAFALLFGILDAFFWPASGSIVPSIVSKEGLTRANSIIQ
TTNQSTSIFGPMISGFVIVALRYEGIFGLITVLLALGALFALLVRVRKSPSPDTKADRIPMLTSVKEGIGYVRQAPVLKA
FVLKTLFLNLFFTGPFAIGIPIFVKHVLQGETLDFSMLQSFLAGGMLISSIVLGFLNLKRRRGLISVLAQLGLSFSFLAL
SMAVTLWTGAVFAACMGISMGISNICAISVIQNHTDKAYIGRVMSIQTMSSMGLTPISYGLTSLLLSVGVGIQNIVMGGA
LILLAFTFWLIVKIPSLRKAD
>lcl|NZ_CP026520.1_prot_WP_042235178.1_5797 [locus_tag=PC41400_RS29365] [protein=helix-turn-helix domain-containing protein] [protein_id=WP_042235178.1] [location=complement(6468352..6468969)] [gbkey=CDS]
MEKQNDIMVLTHLEQLKSLSDPLRIEILCLLVEEARSGQQLSQQMGESRSKVHYHLQDLEKNGLIRLVEENRKGNMTQKL
YRAVAKSFIPSTELLSFHHEYGESRRLMTISTLDRAKSRALAAPESAFLTDSEDHRKWHRIASQTEVNIPKEKFKKWVAK
YHELIQELNRMAEEEVSEEEMETFYLMTVAFQIDEPYFKGEKEDE
>lcl|NZ_CP026520.1_prot_WP_042235176.1_5798 [locus_tag=PC41400_RS29370] [protein=NAD(P)H-dependent oxidoreductase] [protein_id=WP_042235176.1] [location=complement(6469152..6469700)] [gbkey=CDS]
MKIAAVIGSNRKNSYNLKLAEYVQKTFKDRLDVEIVRVNDLPFYNEDIELTPDQSVLDFRGKIAGADAVLWVTPEYNYSI
PGVLKNALDWLSRGDRVMIGKPSWIMGASMGLLGTVRAQEHLRDILFSPGVSSVLLPGNEVYVGLVHEKIDKDGNLHHEP
TIEFLNVVVNNFINWHGKLHKN
>lcl|NZ_CP026520.1_prot_WP_042235174.1_5799 [locus_tag=PC41400_RS29375] [protein=MarR family transcriptional regulator] [protein_id=WP_042235174.1] [location=complement(6469770..6470201)] [gbkey=CDS]
MERDMEEHIGLMIHMADLEITSYLKKWLAPIQLAPEQHLVIALLLRQEGLSQNEIAERLSKDKSGITRMLASLENKGFIR
RTPSAGDRRSVEVYLTDEGRALKATVEEISSHMRQHLRAGFTAEEAAELKRLLTKVRENVRKR
>lcl|NZ_CP026520.1_prot_WP_241688757.1_5800 [locus_tag=PC41400_RS29380] [protein=GNAT family N-acetyltransferase] [protein_id=WP_241688757.1] [location=6470486..6471025] [gbkey=CDS]
MNRFGSPEWDDRPTDQLTGTSEEIDENERYQPTVELKPIDENNWYECTKLEVSDEQKNVFPVPAVYWLAESAYCGFTPLA
VYTGEELIGFAVYAVDPDDGSYWIMAYMIDHKFQHKGLGRSGMEELVRYIHDKHGCDKIVLGHRAENERASRLYASLGFE
EVGRDGIEIIRELTLSKAR
>lcl|NZ_CP026520.1_prot_WP_042235170.1_5801 [locus_tag=PC41400_RS29385] [protein=hypothetical protein] [protein_id=WP_042235170.1] [location=6471209..6471634] [gbkey=CDS]
MVRILDKAAVQPLRRFNLANSFTIRRSPARSRLATITIVIPRNSQPNRVDLVATVGVRGVTGIAQILFRIFRDGKEIFNT
QQGIESIGSEQNYAVTFQAEDRNVKTGTHVYTVTAENRTANTRADVVGPISFSGLAVKTRT
>lcl|NZ_CP026520.1_prot_WP_241688926.1_5802 [locus_tag=PC41400_RS30505] [protein=hypothetical protein] [protein_id=WP_241688926.1] [location=6471807..6472145] [gbkey=CDS]
MEKLAREVIVVHRRSEFTAHEQPVAQMRSFAKVMTPYTISSLHGTADRIERVMLAHAGTGASDVTEVDEVVVSHGYDRDS
GNLVKLGTGTCRVRNHRRCENADEPAGHLRRG
>lcl|NZ_CP026520.1_prot_WP_042235168.1_5803 [locus_tag=PC41400_RS29395] [protein=alpha/beta hydrolase] [protein_id=WP_042235168.1] [location=complement(6472294..6473181)] [gbkey=CDS]
MNKSIRQHQTKDGFILEYSIVGQGEPVLVLHGGHSSCYGELGRNELTARHFSVITPSRPGYGRTSKELGESLIVACDSYI
ELLDELQIPQVNVIAMSAGGPSGIHLASRFPNRVRSLVLQSAVTQCWLNPDDKLYKFSQFLFRPPIEKYVWTIIRLLNKL
FPSFLLSNMIPSFSKLPKEEVLPQINDADRRKFKSIIAQQRSGHGFLIDLVHTGKDATSALAAIKCPTLIMHSIHDASVS
VDHARHASRLIPNSRLCELDLWGHLIWLGTGADEMNRQLFSFLEAANDHQQSANS
>lcl|NZ_CP026520.1_prot_WP_042235166.1_5804 [locus_tag=PC41400_RS29400] [protein=TetR/AcrR family transcriptional regulator] [protein_id=WP_042235166.1] [location=complement(6473239..6473820)] [gbkey=CDS]
MVQVLKEELRQAVIKSAQVEFAQFGYAGASMKRIAAQTGISVGNLYRYYKDKEDLFNSVISPVYHELETLISNHHAQPHG
QGNIFELIVEALTNIVVEYRTPLLILIDGARGTRNENAVQKFHQMMADNVAMHLADYSNKGGEGLKPQVAWPVSVAFLQG
YFEIIRLHEGTEDCKRMVRQYLLVWFQGLQAIL
>lcl|NZ_CP026520.1_prot_WP_042235164.1_5805 [locus_tag=PC41400_RS29405] [protein=BtrH N-terminal domain-containing protein] [protein_id=WP_042235164.1] [location=complement(6474229..6475215)] [gbkey=CDS]
MPMLIEGFALFQGEHCETTAMGHLLQHKGVRLSEPMLFGIGQGLGFIYWDSKGMDFPFIGGRVKPDELSACIASRLNLNI
RVQETSSVTKAWQNVQSCIDRGIPVGLKLDSYYLDYFTNKIHFAAHYAVIYGMDDEYAYMADTSQQGGLVTTRLSNLTMA
RNAKGPMSSRNRSFTIEPTETLPPLAPAIRSSLINNAHDYLNPPIRNMGNKGIKKMSGEILNWPSRSKDLEKDLCLTALL
MERAGTGGALFRNLYRDFLKECVDLFGDSKLEQAYFLFTEIAPMWNGVSASIDNAGRTGNQQELRLASELLVDIAEKERL
AMEFVSTI
>lcl|NZ_CP026520.1_prot_WP_042235162.1_5806 [locus_tag=PC41400_RS29410] [protein=helix-turn-helix transcriptional regulator] [protein_id=WP_042235162.1] [location=complement(6475199..6475771)] [gbkey=CDS]
MNSLSNVEFMLLQIIAESKQASGYQINKLIELRGYREWANIGTTSIYVGLQKLKDKRLIRSEDSVHKSGKGPVPSTYVMM
EAGIETLRNEIIVSLASTRERDNRFDLGLAALPFIGKNEAVEALRKRQNFLEETLRNLGQKYESQGGYRLPLHVRALFRH
PMSLIETEKTFVANLISELTGELEGHAHVD
>lcl|NZ_CP026520.1_prot_WP_042235160.1_5807 [locus_tag=PC41400_RS29415] [protein=sulfurtransferase] [protein_id=WP_042235160.1] [location=complement(6476100..6476936)] [gbkey=CDS]
MRNIVSLEWLYEHLQDPDLLIADCRFALGEPWSGRQDYSIDHIPGAFYVDLEEDMSGEKREHGGRHPLPDLGAFSMRASA
IGVDASKTVVAYDDQGGAMAARLWWMLRFLGHERVYVLDRGYSQWKAAGYPVTAEAPQASPATFSPKVQRHMLASMDEVK
AKLGEKGTVLIDSREGVRYRGEQEAIDPVAGHIPGARNRFWKDALDAEGSWRSAEEQKERFSGLENAEELIVYCGSGVTA
CPNVLALEEAGIPGVKLYAGSWSDWVSYADNPVATGEE
>lcl|NZ_CP026520.1_prot_WP_042235158.1_5808 [locus_tag=PC41400_RS29420] [protein=GNAT family N-acetyltransferase] [protein_id=WP_042235158.1] [location=complement(6477063..6477533)] [gbkey=CDS]
MSRVEIREAELNDRDALRGLMNTYIYDFYHYPRPEEEKLENLLDVLMEKKAGTQFVAVYDGELVGFATLYFTYSTLRTGK
VVIMNDLFVTEKCRIQGVGTKLFRACNQYAAYNGYVAMLWETSKANENAQKFYEKMGGEREEWYPYSIKPVIGVGD
>lcl|NZ_CP026520.1_prot_WP_042235156.1_5809 [locus_tag=PC41400_RS29425] [protein=MarR family transcriptional regulator] [protein_id=WP_042235156.1] [location=6477778..6478221] [gbkey=CDS]
MPDQGYLEMEKSFRQLLRKLSNEWTKYVDRSFSKSQFLILETLVKEGPQKVSSLAEALFITAGAITGMSDKLIAGGYAER
IRTEEDRRVVFLAVTEKGREMVKVMSEYRLGIIKRMMQGLTDEDMTNMSRIFAHMIDNLDKGEEAAD
>lcl|NZ_CP026520.1_prot_WP_042235154.1_5810 [locus_tag=PC41400_RS29430] [protein=GNAT family N-acetyltransferase] [protein_id=WP_042235154.1] [location=6478486..6478977] [gbkey=CDS]
MTLATSWRVRPFTETDGKAICTWSYPAPYSVYNWPSWEELSARAEEFADPDIREKQYAAIVDERDELCGFVQFFPLTGVT
RLGLGIRPDLCGQGLGPSYVECLVAEARRRNPGNEIDLEVLVRNERAIKAYIRAGFAVTDTYERMTPTGMAMMHCMVHGA
EPA
>lcl|NZ_CP026520.1_prot_WP_042235151.1_5811 [locus_tag=PC41400_RS29435] [protein=molybdopterin-dependent oxidoreductase] [protein_id=WP_042235151.1] [location=complement(6479214..6480599)] [gbkey=CDS]
MERKNNWLSAGFGKKLNRLHAWNAWVTLALALTGIVLSIGWIRGDLGALRVALKQGHIWLGVLSIFLLALYVPLLPKHLG
QLRKRRKAGQQGNLAGVLVFTAGWIVSGVILWQFRSLPPVWSNTALFVHDAFTWVGVPYLLYHSITRSRWVKKRSAGRPA
SASAPSPPARAAEAARPQADAPQPILRADTGAIFRDETRDAGGAAEQPSAWSDSLGRRMTRGAFLKLGAAAALAVVIGPS
FVRWARGNLFGGGGLEEAVRADANRMLPAPVPLAESSPPVGGGAKGDFRIYTVTEIPAFSSDNWAFSVSGMVDKPLRFTW
EELQQVERLVQVSDFHCVTGWSVNSVTWEGIRLSELLRQAGVRPGAGFVKFISGDGVYTDCLSLEQARMDEVLVALLMDG
KPIPQKLGGPVRLIVPSMYAYKSVKWLQSIELIRDEHIGYWEVRGYDKDAWVPASRLKKGV
>lcl|NZ_CP026520.1_prot_WP_042235149.1_5812 [locus_tag=PC41400_RS29440] [protein=4a-hydroxytetrahydrobiopterin dehydratase] [protein_id=WP_042235149.1] [location=complement(6480750..6481049)] [gbkey=CDS]
MAAANKLSYDQVAVKLAGLNGWNLRDERTIEKTYECRDFKTAIDFVRRVADVAESLNHHPFIKVDDRRVTMQLSTHDAGG
LTDLDFTSAVQYDGVLPVA
>lcl|NZ_CP026520.1_prot_WP_164977161.1_5813 [locus_tag=PC41400_RS30035] [protein=ATP-binding protein] [protein_id=WP_164977161.1] [location=complement(6481070..6482371)] [gbkey=CDS]
MPSVITVSCITFIFHMHISWIEITYIFTVILFIFSSFAYSMEYFTTKLEKFMFPRKYFLQQSLKKISKALGTITNFNALK
EIILNDIVTTLEIHGGAIVFVEKDGIQVITAGDIDEEDVKTSMENGVYKGDKYTAFEINRHEEYRSFLVVTGKKSNTHLG
QEDRHWINLIVTYLSASLENVYLIRKLTLKMYELMANIPNEEATPEFVWLRKTMYDIQEKERIRIAMDLHDTTMQDLFLL
KRRLYPILQRFPKMTDEYGQLTGILTHIDLINENLRQSCFELNPYVLQKTGLIGSIRKLVDVESSYSEFQVNFIADCTAQ
IELLEVERKKHLFRVIQELLNNAKKHSQAEHVTIRLVEENSSINLFYEDDGIGFDRQLKEMNRKNRIISGAGIGLDQMQG
RVLLMNGEFYMDTERGKGVQIEIRIPFLEGISA
>lcl|NZ_CP026520.1_prot_WP_164977162.1_5814 [locus_tag=PC41400_RS30040] [protein=hypothetical protein] [protein_id=WP_164977162.1] [location=6483479..6483634] [gbkey=CDS]
MDQTLILWSPELKKEIRSEKLKNANSTARLSVEEILKQQLEKNTIYKLAWC
>lcl|NZ_CP026520.1_prot_WP_042235144.1_5815 [gene=recQ] [locus_tag=PC41400_RS29450] [protein=DNA helicase RecQ] [protein_id=WP_042235144.1] [location=complement(6483978..6485777)] [gbkey=CDS]
MLAKAQELLQQYYGYPEFREGQKRIISALLEGRDTLGIMPTGGGKSICYQIPALMMEGVTLVVSPLISLMKDQVDALHTV
GIPATFLNSTLEAGEAGRRIREAMQGKYKLLYIAPERLESDWFRQWMHSMNISFVAIDEAHCVSQWGHDFRPSYVNIAPF
VQDLPSRPLVAAFTATATTEVTQDIARLLELDDPEVVVTGFGRSNLELSVLRGVNKRDFVFDYAKAHTHQPGIVYAATRK
EVDELYEQLRRKGLAAGHYHAGLSDEERARNQEAFLYDDVRVMVATNAFGMGIDKSNVRYVIHYNMPKNMEAYYQEAGRA
GRDGDPSECMLLFSAQDILTQKFLIEQNQLSQDRKSNEYKKLQTMIDYCHTPRCLQNYILQYFGDEGELVPCGNCSSCKD
DRETVDITVEAQKIFSCVRRMQERFGISMVAGVLKGSRNKKVLQYKFDNLPTYGIMKDRTEKEIADLINVLIAEGYMGLS
EGQYPVVRLQAPAVEVLQGRVQVFQKVTPSQPLESAVDNMLFEQLRLLRRDMAAKEKVPPYIIFNDSTLREMSERCPTTE
AQMLKIKGVGEVKFRNYGLPFLELLQEYVREQEAYAGEF
>lcl|NZ_CP026520.1_prot_WP_042235142.1_5816 [locus_tag=PC41400_RS29455] [protein=hypothetical protein] [protein_id=WP_042235142.1] [location=complement(6485852..6486151)] [gbkey=CDS]
MGIKFGREYEDIIRDLMNGLGHVDGFYGFFEMSETDWHELPEEDRQECLRTLADDVFYGLGSEPLLQIADAKVRYLKDRH
LIQIDHDDQVVTIVRLTSA
>lcl|NZ_CP026520.1_prot_WP_042235140.1_5817 [locus_tag=PC41400_RS29460] [protein=SWIM zinc finger domain-containing protein] [protein_id=WP_042235140.1] [location=6486573..6488162] [gbkey=CDS]
MLELQIPKNRMNYLIKQMQTHMDPEDMERGWLFHHKGRVSGVELVGGTVVHASVRGNDRYHTTIDLESFSAGDCTCSFDK
PCRHMAAVIFALYVPHGRPELLVQELKRAMTTKRKSAKNADRESKRRTSEPVPGGSPEDWHRFFEAQYHGFSISHQYSFD
LFVTSAFERFQAFASGWPDGIRLLYDLHVNFFVMHRMEQFFGDTKNSYLSGYHESNAKTAAADCLSRLDRLLASPALPVS
VKEHPKRWTEALNRLASLALQGKEGPNDWLYAYRAVWWALPKKAEWIKREKERLDKQLQEREESRRNRKQDVLLVARAHF
DLMENNADAAFLRLEALHQRRARDFILYMRRCHERAEWDILLSWLRWLAPTMQSAAQDDFRLLCQYWTDAAKHADADDEW
VRIMETLLPRSYYVYTGYLLQTKRYRQWVDLQIANRIGPQNLYALELRSVEESDPSLLLPLYHQAVERSVLEKNRTSYKT
AVRLLKKLHGYYTTLGRKREWETYINRLSDHFVRLRAFQEELEKGSWLP
>lcl|NZ_CP026520.1_prot_WP_042235138.1_5818 [locus_tag=PC41400_RS29465] [protein=DEAD/DEAH box helicase] [protein_id=WP_042235138.1] [location=6488159..6491041] [gbkey=CDS]
MIETSPLTVHAVWMPSGAFFVWGTRENGGIWDAQDLKGLLFAWHAPSFYGTFIETTEWGHREGLSLPALTALDFFADPHT
VKHLSIRWDTDTANLRRLAPEVKEALAGGRFVPDYEQWKAGTLSWKLQLEGEAAASVTSQAELWLDQLIPAWIEADGSRK
AALRRLEEAHPLLRLGTRPADTWLDEEDWLVSIGWKHDPTPFRSSLQLMEPEGDPDWRLQILLQDRSDDTILRAVTPAGE
PLEGELALPESWAADLGRVPRDVGRWLRLLPWLNAADEGEPPQLRTRLSDEEAWRFLAEGSVRLVEAGFAVMLPAWWERA
RKLRPKLQAKLKNSLGGGGGESMFGLTQLMQFDWRVAIDDFELSEEEFLQLLEEKRKLIRIRGRWVQLDPQFLRQAEELL
ARADRQQGLTLREVLELQLLGGGTGGEPEGPADEAADPDNRPPAVEIELTGQLRRLFQQLTRTAELPRIDPPASFLGTLR
PYQLEGSTWLLFLRRFGLGACLADDMGLGKTIQYITYLLQVKETEAPEAPSLLICPTSVIGNWQMELRRFAPNLRVHVHY
GPQRAKGSFFAESVEGADLVITSYTLSHLDEEELREVSWSCICLDEAQNVKNPYTKQAAAIRSFSGLHRIALTGTPMENR
LTELWSIFDFLNPGYLGTLTQFTRRFVTPIEKSRDSGLISGVQRLVRPFLMRRIKKDPAIQLDLPEKNESKTFVTLTAEQ
GTLYEHYIRDLFEKVDAMSPMERRGHILAALTKLKQICNHPALIMREGPRTDWRNRSGKVERLLDMVQELRQQGDKCLIF
TQFVETGHLLQEILQSELGESVSFLHGGTPRQTRDAMIARFQNAELPLEEQFSVFLLSLKAGGTGLNLTAANHVFHFDRW
WNPAVENQATDRAFRIGQTRDVQVHKFVTLGTLEEKIDEMIERKLGLSEQIVGSGENWITELSTDELRDLLALRMEWMNG
>lcl|NZ_CP026520.1_prot_WP_042235136.1_5819 [locus_tag=PC41400_RS29470] [protein=ATP-binding protein] [protein_id=WP_042235136.1] [location=6491663..6494542] [gbkey=CDS]
MKFNRSRLSLSYVIAIVLLLALVGTSSYISSLNMERQSDAVIENAIPVISAANNLLQDLLNQETGIRGYLISSDPKYLEP
YENGKAALQKDLEVIRTLESGHDDLRILIQTQVNPHIQKMLTFYTSQIELVQSGKVQEAQGRINIGKQYMDQYRTLHLSI
LDVLNKISDSSHDSIKRAGAVSRMIILSGGILGILIGIWSGLIQLRASRAESQLRKSEETYRFMAESLEAQNEEIVAQQE
EQQITLAKLSDRERELELISNYQEKLTGFTELKQFLQHSIPALLDSVSYDAALVVLKCKEAEHYEVLFSSGYPRSFASRA
QSELYGASQRVFDEGCAFVQKREATEAERGFHDGISVAVDHYFPLQDDRQQVIGFLLLTNFHGFDRHEEKERLTLGLIRQ
FDLAFYAQIMNDERRKQAAHLEELNDQLVLEKQYIESQRDLIEGILEAAHEGMMMCDRNGKILIANNQMHKFFHLDEQIG
HDFVDSCQEIAASSSGFTGVSQSIKALFDGDINQLNERFTFMNGEEPQFTEIYATSVGENDSDESGYLFVFRDRTEEERI
DAMKNEFVSIVSHELRTPLASVLGFIEILIHRELPREKQQKYLDTIFKEATRLANLINDFLDLQRMESGRQVYHFAPVDM
KELVYDIVEKWQDKQNHDIGISVLSPAEVWVRGDSDRLRQVVQNLLSNAIKYSPGAERIDLQIRESGGSVFIDVTDYGLG
IPEEAKEHLFSKFYRVDNSDRRQIGGTGLGLAIVKEIVEGHQGKLTFRSQMGKGSTFTVELVPFEMVSLNNKIVIFEDDD
NLAKLIQVALAKLQLPSVQLRSAEEGILSLKRKTTEAPLLFIVDIHLEGSLTGWDFISELYHHPVYNRTPVIVSTALEPP
LDYHEKDIEKFMRKPFTMERLIQVAQDLLSNRDKPSYVFPSQDRNLITSSLERNGIKVHDVRVNADTIEFEIETEDEEK
>lcl|NZ_CP026520.1_prot_WP_042235134.1_5820 [locus_tag=PC41400_RS29475] [protein=methyl-accepting chemotaxis protein] [protein_id=WP_042235134.1] [location=complement(6494898..6497063)] [gbkey=CDS]
MKKEMLTKSFKGLNRNSLKVKIPLLLLILALVPLLVSEYFLSNQFTSVLRKSIEENQVKITTYTADYISSTLVQKVSTVQ
SVIQNHTEMGTQTEAQKIAILKTMSENDTEVKIALFADKDGQAVDQNGKKSSLANDETFKSVKDTKKRVVSDIIKDSSSN
QYSLKVSAPVIGANGEFAGMLQTSINPSRMINQVNKVKIGETGYAYLMSKEGIYMVHPQTDRIGKPLSDMATPERLELFK
KTVLVKEQGQVAYKEKDNTNKAAVYEEIKVPGWRIVVTGNSDEIYSQVDQVNFVAKIIVLVSAIVVVAAAIWLSTFILKP
LLGISAMMKRIAGGDLSRRLKVQSQDEMGDVQQNINEMLNSFSALLQKLSMAIEHMAASSQQLSAIATQSADVSQNVNQT
VLVMVKGAETQFETAEQTTEAMNEMAAGVTKIADSASTVSEVAQGAVQRVNQGSRDMQDAIHQIQQASEAVEATSLTIKA
LEESSARIHQVVGFISEIAKQTNLLALNASIEAARAGEHGRGFAVVAQEVKNLAEQTSGATVQINDTIKQVLSATKQAAA
SMSHGKAEVDRGVEQVRNTDRVFEDILETVSSINGQIQEVSAATQQISAGTEEVAASMQETVQISRDSLGKMNDMSRSTE
EQHAGMKEIVISASELNRLASELQELSGKFTVERNAAEDNETEDKEAGNKKIKQPKQKRQKPDKPIKGAKKGRAGKGNTE
K
>lcl|NZ_CP026520.1_prot_WP_042235226.1_5821 [locus_tag=PC41400_RS29580] [protein=pro-sigmaK processing inhibitor BofA family protein] [protein_id=WP_042235226.1] [location=complement(6504618..6504881)] [gbkey=CDS]
MKTAVWGMLALSAGALLVLLVRQPGARKIFSSIGVHVVVAAFLLYGVQLLSGYSRFELPINIYTVGTVSVLGVPGLMLLA
ALKVVLV
>lcl|NZ_CP026520.1_prot_WP_042235223.1_5822 [locus_tag=PC41400_RS29585] [protein=DUF2508 family protein] [protein_id=WP_042235223.1] [location=complement(6504878..6505153)] [gbkey=CDS]
MNDLRKSAFSFQRGTRKTAEQDKQELLAEIRLAHSQWKTAQHHFEHALEKDEIDYAIYAVEAAEKRYEMLLRQAKKLNVT
SAYHITAEVRG
>lcl|NZ_CP026520.1_prot_WP_042235221.1_5823 [gene=recR] [locus_tag=PC41400_RS29590] [protein=recombination mediator RecR] [protein_id=WP_042235221.1] [location=complement(6505323..6505919)] [gbkey=CDS]
MHYPEPISKLIDAFTRLPGVGPKTAGRLAFHVLRMKEEDVLDFAKALVNVKRNLHYCSVCCNITDIDPCRICQDKSRDAS
VICVVQEPKDLVAMERTREFDGYYHVLHGAISPMEGIGPDQIHIAELLKRLGDEQVKEMILATNPNIEGEATAMYLSRLV
KGFGLKVTRIAHGLPVGGDLEYADEVTLTKALEGRREI
>lcl|NZ_CP026520.1_prot_WP_009671602.1_5824 [locus_tag=PC41400_RS29595] [protein=YbaB/EbfC family nucleoid-associated protein] [protein_id=WP_009671602.1] [location=complement(6505982..6506293)] [gbkey=CDS]
MNNMNQMMKQVKKMQEQMLKVQEELGSKTVEGTAGGGVVTVTANGHKKVTAIAIKPEAVDPDDVEMLQDLVLTAVNDALN
KAEELAGKDMGKFTGGMNIPGLF
>lcl|NZ_CP026520.1_prot_WP_042235218.1_5825 [gene=dnaX] [locus_tag=PC41400_RS29600] [protein=DNA polymerase III subunit gamma/tau] [protein_id=WP_042235218.1] [location=complement(6506329..6508068)] [gbkey=CDS]
MTHIALYRTWRSQAFRELVGQKPIVQTLQNSLREQRFTHAYLFSGPRGTGKTSAAKILAKAVNCVHGPAEEPCNECEACI
RISQGSVMDVVEIDAASNRGVEEIRDIRDKVKYAPTEVRYKVYIIDEVHMLTTEAFNALLKTLEEPPAHVIFILATTEPH
KLPATIISRCQRFDFRRVSMEEQVGRLAYVCREENIAIEEEALHYIARLSEGGMRDALSLLDQASSFAVGTIAASDVVAM
TGGLASDQFEKLVGSMKEQDIGAALELIDQFMQEGKSADKCMENLIHYFRDLLMIRLVPDSSVLTERVFDVARLDQVASR
FGTDQLIAIIEVLNQYQSEMKYSVQPQTILEVAVMKACGAIASGGGSRPAEAPPASRSAGEPEPEALSQLTRRIQQLEEQ
LAKLMQGGVPAAGAQQPARSPVAPRSSNAAVRKASVRLDDFTSAVESHHTRALLGKWTQILAMVKERKITVHAWLVDGEP
VAIHGDTVLVAFKSAMHRDTTEKPANKQLIEQVISEALGETLKLATVMQKEWKEAREGASEPAREEMTLQAEDSEPDKEP
WVDEAIKLFGEDLVSIKDE

"""

def comprehensive_protein_analysis(file_object):
    """Parses FASTA data and performs both basic and advanced sequence analyses."""
    # Parse records using Bio.SeqIO
    records = list(SeqIO.parse(file_object, "fasta"))

    if not records:
        print("No protein records found.")
        return

    print(f"Total Proteins Loaded: {len(records)}\n")

    # 1. Length Statistics & Advanced Biochemical Properties Table
    print("=" * 80)
    print(f"{'Header':<15} | {'Length':<6} | {'Mol. Weight':<12} | {'Isoelectric Pt':<15} | {'GRAVY'}")
    print("=" * 80)

    all_sequences = ""
    lengths = []

    for record in records:
        seq_str = str(record.seq).replace("*", "")  # Clean sequence of stop codons
        all_sequences += seq_str
        seq_len = len(seq_str)
        lengths.append((record.id, seq_len))

        if seq_len > 0:
            analysed = ProteinAnalysis(seq_str)
            mol_weight = analysed.molecular_weight()
            pi_val = analysed.isoelectric_point()
            gravy_val = analysed.gravy()

            print(f"{record.id:<15} | {seq_len:<6} | {mol_weight:<12.2f} | {pi_val:<15.2f} | {gravy_val:.2f}")

    print("=" * 80)

    # Sort lengths to find shortest/longest
    lengths.sort(key=lambda x: x[1], reverse=True)
    print(f"\nLongest Protein:  {lengths[0][0]} ({lengths[0][1]} aa)")
    print(f"Shortest Protein: {lengths[-1][0]} ({lengths[-1][1]} aa)")

    # 2. Global Amino Acid Composition & Frequencies
    aa_counts = Counter(all_sequences)
    total_aas = len(all_sequences)

    print("\nGlobal Amino Acid Frequencies:")
    for aa, count in aa_counts.most_common():
        percentage = (count / total_aas) * 100 if total_aas > 0 else 0
        print(f"  {aa}: {count} ({percentage:.2f}%)")

if __name__ == "__main__":
    # To run using a local FASTA file instead, uncomment the lines below:
    # with open("proteins.fasta", "r") as file_handle:
    #     comprehensive_protein_analysis(file_handle)

    # Running analysis on the sample data string
    file_like_obj = StringIO(fasta_data.strip())
    comprehensive_protein_analysis(file_like_obj)

Streaming output truncated to the last 5000 lines.
lcl|NZ_CP026520.1_prot_WP_042231944.1_852 | 193    | 21429.38     | 6.85            | -0.15
lcl|NZ_CP026520.1_prot_WP_042231941.1_853 | 301    | 33339.86     | 5.53            | -0.10
lcl|NZ_CP026520.1_prot_WP_042231937.1_854 | 142    | 14259.83     | 4.27            | 0.33
lcl|NZ_CP026520.1_prot_WP_042231936.1_855 | 128    | 14799.24     | 11.76           | -0.68
lcl|NZ_CP026520.1_prot_WP_042231933.1_856 | 158    | 17953.13     | 5.94            | -0.52
lcl|NZ_CP026520.1_prot_WP_042231931.1_857 | 183    | 19724.53     | 5.96            | 0.01
lcl|NZ_CP026520.1_prot_WP_042231928.1_858 | 313    | 34705.03     | 5.25            | -0.28
lcl|NZ_CP026520.1_prot_WP_042234209.1_859 | 397    | 44093.62     | 5.31            | -0.15
lcl|NZ_CP026520.1_prot_WP_009674668.1_860 | 68     | 7896.16      | 6.25            | -0.32
lcl|NZ_CP026520.1_prot_WP_042234207.1_861 | 66     | 7290.06      | 4.94            | -0.37
lcl|NZ_CP026520.1_prot_WP_04223